# Split-K matmul for skinny outputs

When $M$ and $N$ are small but $K$ is large (a decode-time projection with a handful of tokens, say $16 \times 4096$ times $4096 \times 64$), an output-tiled matmul launches only a few programs and leaves most of the GPU idle, each program grinding through all of $K$. **Split-K** also cuts the reduction dimension into `split_k` slices, computes one partial product per slice in parallel, and sums the partials afterwards.

Write `split_k_matmul(a, b, split_k, BK)`.

## Inputs
- `a`: `float32` CUDA `(M, K)`; `b`: `float32` CUDA `(K, N)`; all sizes $\ge 1$. Use their strides.
- `split_k`: Python int $\ge 1$. `BK`: Python int, a power of two $\ge 16$ (the K step inside a program).

## How K is split
Every slice has the same length, rounded up to a multiple of `BK`:

$$\text{chunk} = \left\lceil \frac{K}{\text{split\_k} \cdot BK} \right\rceil \cdot BK, \qquad \text{slice } s = \bigl[\, s \cdot \text{chunk},\ \min((s+1)\cdot\text{chunk},\, K) \bigr)$$

Trailing slices can be empty (their partial is all zeros).

## Output
A tuple `(c, partial)`:
- `partial`: `float32` `(split_k, M, N)`; `partial[s] = a[:, slice_s] @ b[slice_s, :]`.
- `c`: `float32` `(M, N)` $= \sum_s$ `partial[s]` (that is, $a b$).

## Requirements
- A 3-D grid `(cdiv(M, 32), cdiv(N, 32), split_k)` with `BM = BN = 32`: program `(pid_m, pid_n, pid_k)` computes its output tile over slice `pid_k`, accumulating in float32 with `tl.dot(..., input_precision="ieee")`, and stores it to `partial[pid_k]`.
- Mask the K edge as well as M and N. Reduce the partials with a fixed-order sum (no atomics).

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def split_k_matmul(a, b, split_k, BK):
    """a (M, K), b (K, N) float32 CUDA. Return (c (M, N), partial (split_k, M, N))."""
    raise NotImplementedError


@triton.jit
def _split_k_kernel(a_ptr, b_ptr, part_ptr, M, N, K, chunk,
                    stride_am, stride_ak, stride_bk, stride_bn,
                    BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0], [2.0, 2.0, 2.0], [3.0, 3.0, 3.0], [4.0, 4.0, 4.0], [5.0, 5.0, 5.0], [6.0, 6.0, 6.0], [7.0, 7.0, 7.0], [8.0, 8.0, 8.0], [9.0, 9.0, 9.0], [10.0, 10.0, 10.0], [11.0, 11.0, 11.0], [12.0, 12.0, 12.0], [13.0, 13.0, 13.0], [14.0, 14.0, 14.0], [15.0, 15.0, 15.0], [16.0, 16.0, 16.0], [17.0, 17.0, 17.0], [18.0, 18.0, 18.0], [19.0, 19.0, 19.0], [20.0, 20.0, 20.0], [21.0, 21.0, 21.0], [22.0, 22.0, 22.0], [23.0, 23.0, 23.0], [24.0, 24.0, 24.0], [25.0, 25.0, 25.0], [26.0, 26.0, 26.0], [27.0, 27.0, 27.0], [28.0, 28.0, 28.0], [29.0, 29.0, 29.0], [30.0, 30.0, 30.0], [31.0, 31.0, 31.0], [32.0, 32.0, 32.0], [33.0, 33.0, 33.0], [34.0, 34.0, 34.0], [35.0, 35.0, 35.0], [36.0, 36.0, 36.0], [37.0, 37.0, 37.0], [38.0, 38.0, 38.0], [39.0, 39.0, 39.0], [40.0, 40.0, 40.0], [41.0, 41.0, 41.0], [42.0, 42.0, 42.0], [43.0, 43.0, 43.0], [44.0, 44.0, 44.0], [45.0, 45.0, 45.0], [46.0, 46.0, 46.0], [47.0, 47.0, 47.0], [48.0, 48.0, 48.0], [49.0, 49.0, 49.0], [50.0, 50.0, 50.0], [51.0, 51.0, 51.0], [52.0, 52.0, 52.0], [53.0, 53.0, 53.0], [54.0, 54.0, 54.0], [55.0, 55.0, 55.0], [56.0, 56.0, 56.0], [57.0, 57.0, 57.0], [58.0, 58.0, 58.0], [59.0, 59.0, 59.0], [60.0, 60.0, 60.0], [61.0, 61.0, 61.0], [62.0, 62.0, 62.0], [63.0, 63.0, 63.0]], "dtype": "float32"}, 2, 16], "e": {"__tup__": [{"__nd__": [[2016.0, 2016.0, 2016.0], [2016.0, 2016.0, 2016.0]], "dtype": "float32"}, {"__nd__": [[[496.0, 496.0, 496.0], [496.0, 496.0, 496.0]], [[1520.0, 1520.0, 1520.0], [1520.0, 1520.0, 1520.0]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float32"}, {"__nd__": [[1.0], [10.0], [100.0]], "dtype": "float32"}, 1, 16], "e": {"__tup__": [{"__nd__": [[321.0]], "dtype": "float32"}, {"__nd__": [[[321.0]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.10468166321516037, -0.8065453171730042, 0.8919731378555298, -0.9285507798194885, -1.1356676816940308, 0.5134580135345459, 1.319050669670105, 1.2758177518844604, 1.3556002378463745, 0.40296003222465515, -0.34895575046539307, 0.5779352188110352, 0.5794493556022644, 0.0661078616976738, -0.6986229419708252, -0.17757895588874817, 0.26550811529159546, -1.155428171157837, -0.36372479796409607, -1.2169758081436157, 1.7756662368774414, 0.7107280492782593, -0.14065319299697876, 0.9527892470359802, 0.17559249699115753, -0.5476267337799072, 0.028777964413166046, -0.1313570886850357, 3.641681671142578, -0.9858279228210449, -2.7848196029663086, 0.7665310502052307, -1.3263137340545654, -1.3989546298980713, 1.3564752340316772, -0.8927116394042969, -1.3230607509613037, 0.01835167594254017, -1.2653939723968506, 0.7417453527450562], [-0.7936019897460938, 0.09403914213180542, 1.506269931793213, 2.020174980163574, -2.2683587074279785, -0.5005839467048645, -0.4062609076499939, 0.8529298305511475, -0.43904149532318115, -0.9312676787376404, 1.5131419897079468, 1.3407573699951172, 0.13185586035251617, -2.1364974975585938, -0.14161764085292816, -1.4062128067016602, 2.0727298259735107, 0.5134504437446594, 0.3769960105419159, 0.8980483412742615, 0.7343261241912842, -1.2340610027313232, 0.634585976600647, 0.3554089367389679, -2.610100746154785, 1.208814263343811, 0.4471503794193268, -0.0946047306060791, 0.5522430539131165, -0.8403838872909546, 1.0841730833053589, -0.3856477737426758, -1.293161392211914, 2.119006872177124, 0.8554474115371704, 0.08914987742900848, 0.30090832710266113, 0.5626627802848816, 1.538124680519104, 0.7753117084503174], [1.4379879236221313, -0.6563751697540283, -0.7444276213645935, 0.25438451766967773, 0.4786055386066437, -0.028811965137720108, -0.8431254625320435, -0.2960953414440155, 0.9279054403305054, 0.8590631484985352, -1.1776049137115479, 0.8619683980941772, 0.49836277961730957, 0.40042442083358765, 0.9738825559616089, 0.3124775290489197, 1.8074710369110107, -0.491681307554245, -0.9381097555160522, 0.3162057101726532, 0.498546838760376, -1.062974452972412, 0.5288833379745483, 0.3239125907421112, -1.1330124139785767, -0.29039397835731506, 1.1363911628723145, 1.0072283744812012, -1.0945528745651245, 0.8100159168243408, 0.08211134374141693, 2.198730707168579, -0.18404521048069, 0.66422039270401, -1.5809545516967773, 1.1641223430633545, -0.015859873965382576, -0.548403799533844, -0.49963244795799255, -0.14341841638088226], [-0.573279857635498, 0.6214244961738586, -0.4837692379951477, -0.2837143838405609, 3.0221776962280273, -0.045127883553504944, 0.511593759059906, 0.31080323457717896, -0.05607115849852562, 0.1421217918395996, -1.3477555513381958, 0.7503446936607361, 0.2481265515089035, 0.1531074196100235, -1.1415971517562866, 0.9608229994773865, -1.4083664417266846, -0.09069685637950897, -0.7007060050964355, -1.4378266334533691, 1.5623375177383423, 1.4338330030441284, 1.2869844436645508, -0.006138356868177652, -0.8896258473396301, 0.1835116595029831, -0.14141488075256348, -1.8243428468704224, 1.2042388916015625, 0.8366813659667969, -0.6325790882110596, -1.0399388074874878, -0.07576574385166168, 0.21080642938613892, 2.196359634399414, 0.4142724275588989, -1.0330719947814941, 0.6605967879295349, -1.2315905094146729, -1.052594780921936]], "dtype": "float32"}, {"__nd__": [[0.5111566781997681, -0.22079677879810333, -1.4502513408660889, -0.7081471085548401, -0.48418334126472473], [-1.0513579845428467, -0.4927869439125061, 0.1285572350025177, -0.2700670063495636, -0.880038321018219], [0.7451338171958923, 1.5828863382339478, 0.8827695846557617, -1.0951634645462036, -0.8214902281761169], [0.09469897300004959, 0.1831129938364029, -0.9538848400115967, 0.48369696736335754, 0.021739980205893517], [-1.2312356233596802, -1.772970199584961, 0.005742082372307777, -0.6701810359954834, 1.9603848457336426], [-0.949763834476471, -0.5993463397026062, 2.8347551822662354, -1.178462266921997, 1.0377352237701416], [0.19423553347587585, -0.29859885573387146, -1.081479787826538, 0.14411364495754242, 1.579131841659546], [0.8922150135040283, 0.007472758647054434, 0.9403536319732666, -0.35181954503059387, 0.20487911999225616], [1.2150535583496094, 0.09418055415153503, 0.8932214975357056, 0.14734315872192383, -1.050977110862732], [1.3456121683120728, -0.7042168378829956, 1.3721213340759277, 1.1029667854309082, 1.4536974430084229], [0.8066694140434265, -0.15114876627922058, -0.902579665184021, -0.5112901926040649, -0.5131332874298096], [1.0701137781143188, -1.655621886253357, 0.998753011226654, -0.2515871524810791, 0.46067750453948975], [0.9378744959831238, 0.6399665474891663, 1.0816839933395386, -0.6805874705314636, -0.42184802889823914], [-1.3269211053848267, -0.19838397204875946, -0.11144847422838211, 0.41362035274505615, 0.6052985191345215], [1.241477131843567, 1.3237777948379517, -0.5707613825798035, -0.18363605439662933, -0.38186416029930115], [-0.28560498356819153, 1.1403928995132446, 1.5626875162124634, 0.7494915723800659, 0.5809842348098755], [-0.8456148505210876, -0.5039567351341248, -0.08726052194833755, -2.330630302429199, 0.4164133667945862], [-0.15427318215370178, 1.8573768138885498, -0.11370290070772171, -1.6718770265579224, 0.14722216129302979], [-0.1335310935974121, 0.28642329573631287, -1.1096338033676147, 0.05606847256422043, -1.7908542156219482], [0.39611345529556274, -0.14989669620990753, -2.2803027629852295, 0.7358688116073608, 1.1031534671783447], [1.1837184429168701, 0.7322801351547241, -0.5860660672187805, -0.7147589921951294, 0.5303341150283813], [1.1479816436767578, 2.82061505317688, 0.3552108407020569, -1.7482285499572754, -0.3092803955078125], [0.8671424984931946, 0.6636236906051636, 1.0781539678573608, -0.8887432813644409, -0.4095929265022278], [0.7181432843208313, 0.4380524754524231, 0.417124480009079, -0.5995995998382568, -0.47020837664604187], [0.5284940004348755, 0.9218194484710693, -1.2731924057006836, 0.2241811901330948, 0.602205753326416], [0.3198188245296478, 0.35104963183403015, -0.07710755616426468, -0.7346887588500977, 1.1738203763961792], [-1.276419997215271, 0.29217827320098877, -1.1644463539123535, -0.6373396515846252, -0.14950720965862274], [-1.0626459121704102, 0.070877306163311, 0.8445547223091125, 1.1845312118530273, -0.4643956124782562], [0.2943641245365143, -0.5066274404525757, -1.1367496252059937, -0.5947785377502441, 2.014265298843384], [-1.052378535270691, -0.6117310523986816, -0.36774393916130066, 0.9766630530357361, 0.10409028083086014], [-0.5775114893913269, -0.6425269842147827, 0.2063571959733963, 0.2009143978357315, -1.8163305521011353], [0.06826216727495193, 0.1507311463356018, 0.3411676287651062, -0.3959938883781433, 0.47215306758880615], [-1.1700217723846436, 0.20828677713871002, 2.767233371734619, 0.42802467942237854, -0.9328165650367737], [0.6026407480239868, 0.9940878748893738, 0.7562634348869324, -0.3287214934825897, 0.12024721503257751], [0.2932545244693756, 3.2393689155578613, -1.8991639614105225, -0.3693033456802368, -0.2717113792896271], [-0.33433184027671814, 0.08749683946371078, 1.1389076709747314, -1.253661870956421, -0.18577660620212555], [-0.986648440361023, 0.6215872764587402, 1.8750722408294678, -1.0035649538040161, -0.1501755714416504], [-0.38370102643966675, -0.9633074998855591, -0.4386160373687744, -0.060737013816833496, -1.2880172729492188], [0.7598475217819214, -0.9382107257843018, -2.1720163822174072, 0.11253824084997177, -0.25931864976882935], [0.6896458268165588, -0.03928720951080322, 0.30010396242141724, 0.533517062664032, -1.2597575187683105]], "dtype": "float32"}, 4, 16], "e": {"__tup__": [{"__nd__": [[14.992993354797363, 6.133930206298828, -3.474095344543457, -5.274378776550293, 12.310808181762695], [10.677499771118164, 2.1411373615264893, -9.686128616333008, -10.816450119018555, -10.125285148620605], [-2.6986894607543945, -9.67963695526123, 10.376806259155273, -1.4683701992034912, 2.6564152240753174], [0.1787896752357483, 4.15054988861084, 4.02968692779541, -5.486159324645996, 12.176681518554688]], "dtype": "float32"}, {"__nd__": [[[5.948747158050537, 1.107820749282837, 6.036431312561035, -1.07967209815979, 0.12256648391485214], [8.812783241271973, 4.21558952331543, -3.0574774742126465, -2.588559865951538, -9.487994194030762], [3.2164218425750732, -1.589181900024414, 1.8430591821670532, 1.1444207429885864, 1.1315025091171265], [-6.4584550857543945, -7.91000509262085, 4.898443222045898, 0.0959058329463005, 9.156229019165039]], [[6.790853023529053, 2.0590436458587646, -1.6716172695159912, -6.2850775718688965, 11.753204345703125], [-2.3370139598846436, -4.861971378326416, -0.33887162804603577, -6.800723075866699, 0.2960965037345886], [-5.4254279136657715, -4.685794830322266, 2.959427833557129, -1.3565884828567505, 0.041093163192272186], [6.811899662017822, 4.797366619110107, 2.7485954761505127, -4.446311950683594, 2.6484954357147217]], [[2.2533926963806152, 2.9670658111572266, -7.83890962600708, 2.0903704166412354, 0.4350373148918152], [4.201730728149414, 2.7875192165374756, -6.289779186248779, -1.4271671772003174, -0.9333873391151428], [-0.48968324065208435, -3.40466046333313, 5.574318885803223, -1.2562024593353271, 1.4838194847106934], [-0.17465533316135406, 7.263188362121582, -3.6173512935638428, -1.13575279712677, 0.37195706367492676]], [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.5364747047424316, 0.9192735552787781, 1.2106959819793701, -0.7380468249320984, 0.08760391175746918, 0.19852450489997864, -1.3605144023895264, 0.590932309627533, 0.4972577393054962, 0.5038295388221741, 0.2191881388425827, 1.0774073600769043, 0.5104904770851135, 0.36230775713920593, 0.8126958608627319, -0.19997847080230713, -0.38948842883110046, 0.8263952136039734, -0.10470566898584366, 0.3211789131164551, -1.4646482467651367, 0.25281956791877747, -1.7518723011016846, -0.6221041679382324, -0.19581711292266846, 2.4262242317199707, -0.2566211223602295, 0.7582154273986816, -0.5624875426292419, -0.307936429977417, -1.6442822217941284, 1.0800663232803345, 0.5630863308906555, 0.7937012910842896, 0.38054803013801575, -0.07341064512729645, 0.6159693002700806, -0.06521376222372055, 1.4636621475219727, 0.567226767539978, -0.352751761674881, -1.1573970317840576, -0.18216250836849213, -1.1056807041168213, 0.4299522638320923, -0.07396101951599121, -1.661865234375, 1.1680203676223755, -0.6310774087905884, -1.8840402364730835, 1.3508014678955078, -0.3295409083366394, -0.5509608387947083, -0.5539348125457764, 0.07061620056629181, -0.09335260838270187, -1.2369695901870728, 0.009768083691596985, 0.013084993697702885, 0.3951994776725769, -1.6142821311950684, -0.3886035084724426, -1.4030207395553589, -0.6307106018066406, -1.2179241180419922, 0.37708139419555664, -0.14228500425815582, -0.15750755369663239, 0.17005737125873566, -0.9108189940452576, -1.3301247358322144, -1.234539270401001, 1.0696539878845215, -0.32135218381881714, 2.4105823040008545, 0.7905035018920898, 0.08174345642328262, 0.8690222501754761, -0.6824533343315125, 0.19018012285232544, -0.035041049122810364, 0.9404796957969666, -1.5313310623168945, -0.9750115275382996, 0.511040449142456, 0.8501442074775696, 1.540849208831787, -0.050916798412799835, 0.13137896358966827, -1.3467803001403809, 0.5204359889030457, 0.6325441002845764, -0.25094446539878845, 0.34878888726234436, 0.21161293983459473, 0.48594745993614197, 0.10489361733198166, 1.4968355894088745, 2.092280864715576, -0.15676814317703247, -0.10533984750509262, -1.2755727767944336, -0.029639912769198418, 1.2838635444641113, -0.9713876247406006, -0.781018078327179, 1.674926519393921, -1.7482104301452637, 1.105424165725708, 0.6243992447853088, 0.030694516375660896, -0.6137710213661194, 0.30934861302375793, 1.3846690654754639, -1.2575045824050903, 1.635642409324646, 0.935398280620575, -0.5211971998214722, 0.008182935416698456, 0.9037092924118042, -1.2529857158660889, -1.5505398511886597, -1.1246784925460815, -1.1535143852233887, 0.11952333897352219, 0.9256125092506409, -0.8046424984931946, -1.4331399202346802, 0.051618561148643494, -0.7243105173110962, -0.3297517001628876, 0.15611572563648224, 0.5881834626197815, 0.23817560076713562, 0.3914220333099365, -0.07532403618097305, 1.2948793172836304, -0.8720994591712952, 2.9856691360473633, 0.07972465455532074, 0.6434237957000732, 0.35276010632514954, -1.1015760898590088, 0.15129421651363373, 0.9535216093063354, 1.0080902576446533, 0.6045935750007629, -0.20919518172740936, -0.6299582719802856, -1.0289759635925293, 1.3683127164840698, 1.5899829864501953, -0.4153262674808502, 0.4414571225643158, -0.165479376912117, -0.014661715365946293, -0.2579784691333771, 1.2471941709518433, 0.2254936248064041, -0.3808791935443878, 1.2829291820526123, -1.388672947883606, 0.5295708179473877, 0.2717956304550171, -0.9577822685241699, -1.7737113237380981, -1.6600924730300903, -0.47839444875717163, -0.755990743637085, -2.0770208835601807, -0.3052842319011688, -0.614726185798645, -0.31076890230178833, -0.3798964321613312, -0.42206257581710815, -0.8086985349655151, 0.23742814362049103, -0.3201155960559845, 1.3099899291992188, 0.0881139487028122, -1.1557451486587524, 2.1612210273742676, 0.3317468464374542, 0.9178056716918945, -0.05774283781647682, -1.2706671953201294, 0.6617989540100098, 0.4929293692111969, -1.7829803228378296, -1.0635671615600586, 0.7350525856018066, -0.5688027143478394, -0.39177191257476807, 0.6036184430122375, 2.1529290676116943, 0.16488562524318695, 1.7414696216583252, -1.2160755395889282, 0.8843459486961365, -0.008245948702096939, -2.1010775566101074, -0.08117183297872543, -0.645389199256897, -0.6562961339950562, 2.7867233753204346, -2.1319637298583984, 0.13287299871444702, -2.0905141830444336, 0.37674590945243835, -0.17339301109313965, 0.9877891540527344, 0.38396990299224854, 0.10706045478582382, -0.6918378472328186, -0.35605019330978394, 1.2406010627746582, 0.89149409532547, -1.1277912855148315, -0.23854972422122955, 1.0470967292785645, 0.8922582864761353, 2.063248872756958, -0.8073171377182007, 0.6653221845626831, -0.2514828145503998, 0.5270697474479675, -0.0007556338096037507, 0.8059741854667664, 1.2608410120010376, -0.7005738019943237, -2.5143799781799316, 0.5987846851348877, 0.03030473180115223, -0.48349907994270325, 0.8494659662246704, 2.3144137859344482, 0.4772166609764099, -0.7221522331237793, -0.320161372423172, -0.48822781443595886, -0.8037633895874023, -0.7024321556091309, -0.5984635353088379, -0.5508480072021484, 0.20769049227237701, -0.24374067783355713, 0.12761612236499786, -1.1320624351501465, -0.8625273704528809, 0.7456511855125427, -1.6820935010910034, -0.023605255410075188, -0.3253891170024872, -0.8326120972633362, 0.04418593645095825, -0.25519657135009766, -1.3629497289657593, 1.0088375806808472, 0.7087444067001343, -2.056272029876709, 0.10337892919778824, 1.1063052415847778, 2.3170523643493652, 1.2930908203125, -1.5029407739639282, -0.5223432183265686, -0.6313387155532837, -0.9182599186897278, -1.6317548751831055, 0.10928165912628174, 0.16366131603717804, -0.487255722284317, 0.4165632426738739, -0.3985348343849182, 1.1430633068084717, 0.6614748239517212, 0.6557782292366028, -0.4421502649784088, -2.037411689758301, -0.37099966406822205, 0.2739964723587036, -2.992950916290283, -0.42882636189460754, 1.9277065992355347, -0.9080404043197632, -0.44982272386550903, 0.9932695627212524, 1.6541045904159546, -0.530976414680481, 1.7587004899978638, -0.0757729709148407, -0.02658357284963131, -0.7783015370368958, -0.013773570768535137, 1.2538964748382568, -1.5252680778503418, -0.3134147822856903, -1.5181059837341309, -0.4879855513572693, 0.04058803990483284, 0.18162228167057037, -0.5428044199943542, 0.40202847123146057, -0.29835018515586853, 0.9574355483055115, -1.6955903768539429, -1.704715609550476, 1.8357759714126587, -1.3781814575195312, -2.7330470085144043, -1.6292424201965332, 0.0033624055795371532, 0.468802809715271, 2.0839829444885254, 2.136002779006958, -0.2260080724954605, 1.2487226724624634, 0.22215671837329865, 0.20735014975070953, 1.0473995208740234, 1.3388820886611938, -0.18966270983219147, 1.3063095808029175, 0.5927284359931946, 0.22325055301189423, -0.4908052980899811, 0.5704984664916992, 1.1109225749969482, 0.5712917447090149, -0.048134855926036835, 1.0400283336639404, 0.46682208776474, -0.8751986026763916, 1.275557041168213, -2.723231077194214, -1.1791741847991943, -1.605234980583191, 0.5422102808952332, 0.18843288719654083, 0.15991508960723877, 0.06183980777859688, -1.5931495428085327, -0.6932857036590576, 1.3097752332687378, 0.5446407794952393, -0.5921136140823364, -1.7616158723831177, 1.053966760635376, 0.3072921335697174, 0.4785695970058441, 0.25150731205940247, 0.5633454322814941, -3.1280057430267334, 0.4177548587322235, 0.7365785241127014, -0.9565303325653076, 1.8995046615600586, -0.5496648550033569, -0.7642958760261536, 0.5289381742477417, -0.5174562335014343, -0.9957733154296875, -0.4552604854106903, -0.6968389749526978, -0.524337649345398, -0.07812348008155823, 0.25273552536964417, -1.620972752571106, 0.5061360597610474, 0.37423449754714966, 1.343759536743164, 0.9135084748268127, -0.6850994825363159, -0.4764286279678345, -0.2838996350765228, -0.1346769779920578, -0.5129191279411316, 0.51651930809021, -0.04702385514974594, 0.24304139614105225, -1.336513876914978, -2.088684320449829, 0.1433308720588684, 0.05988086760044098, -1.66446852684021, -0.6378152370452881, -0.6708289980888367, 0.04170143976807594, -0.16931498050689697, 0.2830716669559479, -1.0673376321792603, -0.3382124602794647, -0.4455571174621582, 0.27410003542900085, 0.3906048536300659, -0.2802351415157318, -1.7583972215652466, -0.20932427048683167, -1.6525171995162964, -0.8883363604545593, -0.38854801654815674, -0.3856944441795349, 2.2428956031799316, 1.444159746170044, 1.3956067562103271, 0.7098613381385803, -1.0122216939926147, 0.7245346307754517, -0.49633216857910156, 0.07886707037687302, -1.4859774112701416, -0.42789337038993835, 1.1303259134292603, -2.3149425983428955, 0.6776478290557861, -0.49064385890960693, -0.7105449438095093, 1.3481594324111938, -0.6676077246665955, 0.8159152865409851, 0.3869756758213043, -0.25420981645584106, 0.3990110754966736, -0.5847289562225342, 1.404101848602295, 0.9579253792762756, -1.4980522394180298, -0.6298633217811584, -0.03554855287075043, 0.07435876131057739, 0.9187221527099609, -0.983961820602417, -1.3671116828918457, 0.06673331558704376, -0.3153831958770752, 0.6341211199760437, 1.670454502105713, -0.8329193592071533, -1.049493670463562, -1.1873140335083008, 1.3047987222671509, 0.8324283957481384, -0.1992729753255844, 0.37046656012535095, -1.2783170938491821, 0.6288012266159058, -0.26093393564224243, -0.30345895886421204, 0.03385864570736885, 0.1680319756269455, 0.08642609417438507, -2.964627742767334, 1.1087850332260132, -1.5251621007919312, -0.8398411273956299, 0.23912771046161652, -0.785703182220459, -1.0574543476104736, 0.10469824075698853, 0.7376207113265991, -1.1420300006866455, -0.2935839295387268, -1.6992006301879883, -0.05259154736995697, 1.97209632396698, 0.9478409290313721, -0.6189083456993103, -0.018977073952555656, 0.090033158659935, 0.03775369003415108, 0.38415250182151794, 1.0024759769439697, 0.43446943163871765, -0.040877778083086014, 0.4497784376144409, -0.46757277846336365, -0.7677901387214661, -1.5298657417297363, 0.5973014831542969, 0.32321590185165405, 0.5609661340713501, -0.4103582203388214, 1.0977182388305664, 0.9290080070495605, 0.2641947567462921, 1.082461953163147, -0.023888874799013138, 0.7961979508399963, -0.7250837683677673, 0.932643473148346, -1.464530110359192, -0.8801568746566772, 0.04014096036553383, 1.0927095413208008, 0.049299608916044235, -1.1128580570220947, 0.4755770266056061, -2.427096366882324, 0.012472515925765038, 0.08550367504358292, -0.12054150551557541, 0.36305972933769226, 0.3028390407562256, 0.03588494658470154, -0.6738379001617432, -0.435993492603302, 0.38722753524780273, -1.360336422920227, -1.1447724103927612, -0.04827500507235527, -0.22447921335697174, 0.6594240069389343, 0.8158895373344421, -0.21176323294639587, -0.6853511929512024, 0.8659071922302246, -1.602799654006958, 0.8035513162612915, 1.7187212705612183, -0.4763312041759491, 0.7732482552528381, 0.500115156173706, -0.6317205429077148, -0.6671774387359619, 0.9716262817382812, -0.12070556730031967, -0.8946854472160339, -2.2892532348632812, -0.26289376616477966, 0.007130964659154415, 0.4367397725582123, -0.10163036733865738, -1.454555869102478, -1.333044409751892, -0.7876719832420349, 0.05502142757177353, -0.6648480296134949, 0.28495439887046814, 0.6659325361251831, 0.29802194237709045, 0.7307451367378235, 0.46112605929374695, -0.2974490225315094, -0.7389004826545715, 1.502607822418213, 0.4180169999599457, 1.0462473630905151, 1.1345136165618896, 0.39014744758605957, 0.3530483841896057, 1.0083993673324585, -0.43383505940437317, -1.1875985860824585, 0.5542805194854736, 1.1033490896224976, -0.6157613396644592, -0.27821290493011475, 0.4104699194431305, -0.859018862247467, -1.492993712425232, 0.2810693085193634, -0.18030640482902527, -1.5153864622116089, 1.0312395095825195, -0.9521855711936951, -1.4516063928604126, 0.0591321736574173, 0.06602871417999268, 1.842269778251648, 0.4187620282173157, -1.163244605064392, -0.9722167253494263, -0.8067201375961304, 0.39658859372138977, -0.5356863141059875, 0.6440862417221069, -0.40668532252311707, -1.8463362455368042, -2.3479154109954834, -1.1148953437805176, -0.6044748425483704, -0.39346733689308167, 0.7349743843078613, 0.5994810461997986, -1.1183840036392212, 0.24279440939426422, -1.767518162727356, 0.9393317699432373, -0.1251707822084427, -1.3522599935531616, 0.8126201629638672, 0.16589181125164032, 0.8412489891052246, -0.30219200253486633, 0.15823400020599365, 1.0180106163024902, -0.9497727155685425, 0.39488810300827026, -1.070603609085083, 3.1795475482940674, -0.04148134961724281, 0.33975324034690857, 1.5945730209350586, 0.005988189484924078, 0.40653282403945923, 0.1888861209154129, 0.9430111646652222, 0.25644397735595703, 0.41236668825149536, 0.23610465228557587, -0.5564737915992737, 0.7421198487281799, -1.1596970558166504, -1.8563940525054932, 0.42180314660072327, 1.5437475442886353, 1.324867606163025, 0.32083144783973694, -0.946444034576416, -0.03135203197598457, 0.3175634443759918, 1.3918039798736572, 0.23436520993709564, 1.5030736923217773, 0.5003064870834351, 0.48340409994125366, -0.16209739446640015, -0.02731047011911869, -0.6600071787834167, 0.016660654917359352, -0.11380375921726227, 1.3104225397109985, -1.1718063354492188, 0.08312415331602097, -0.050207749009132385, -1.0076088905334473, 1.421728491783142, 0.951816201210022, 0.42787325382232666, -0.6334792375564575, -0.6270087361335754, -0.8134365081787109, -2.0019021034240723, 2.126420497894287, 0.8243647217750549, 0.8862961530685425, -0.7214399576187134, -0.8047484159469604, 0.6803531646728516, -0.6526366472244263, 0.8597277402877808, -1.0227115154266357, -0.4308694005012512, -0.33726766705513, -0.22670899331569672, -1.3083815574645996, 0.4641980826854706, -0.02741391956806183, -2.9739866256713867, -1.9940038919448853, 0.06396452337503433, -0.9023783802986145, -0.7528685331344604, -2.0529704093933105, 0.3754860758781433, 0.35248562693595886, -0.14417722821235657, -0.5336189270019531, -0.3718806803226471, 0.9519839882850647, 0.5178886651992798, 0.5824435949325562, -0.12574021518230438, 0.020585067570209503, -0.2775871455669403, 0.7233420014381409, 0.7289493680000305, -1.9616258144378662, 0.6800057888031006, 1.9298303127288818, 0.48619529604911804, 0.025621477514505386, 0.2315710037946701, -0.1221044659614563, 0.9174463152885437, 0.9443029165267944, -0.5379931330680847, -0.5001683235168457, 1.4005813598632812, -1.313825249671936, -0.13662642240524292, 0.8187440037727356, -0.3619335889816284, 0.17222562432289124, 1.5919777154922485, 0.3833082914352417, 1.1751891374588013, -0.4565652310848236, 0.20578806102275848, -2.322219133377075, 0.6268729567527771, -0.5518076419830322, -0.1923539936542511, 1.0071024894714355, -2.109341859817505, 0.1854449361562729, 0.9098367094993591, 0.95663982629776, -2.0233266353607178, 0.5448036789894104, 1.3278546333312988, -0.9851896166801453, 0.48676422238349915, 0.48642972111701965, -0.23134200274944305, -0.19637636840343475, 1.2644795179367065, 0.40314429998397827, -0.180122509598732, -0.9893970489501953, 1.1749540567398071, 2.2566864490509033, 1.4686208963394165, -2.347317934036255, -1.2463041543960571, 0.3099433183670044, 0.14120705425739288, -0.16195844113826752, -2.247169017791748, -0.9106709361076355, 0.49273496866226196, -0.906436562538147, -1.4499353170394897, -0.3061593472957611, 0.2629493176937103, 0.4885641932487488, -0.9669395089149475, -1.1329981088638306, -0.16073377430438995, 0.3042134940624237, 1.0176364183425903, -0.9840990304946899, -0.5231059193611145, -1.8025795221328735, -1.6693426370620728, 1.57660710811615, 0.145736962556839, 0.804456889629364, 0.15919823944568634, -0.7718403935432434, 1.0892024040222168, 1.1700375080108643, 0.4181826412677765, 1.4035879373550415, 0.23203034698963165, -0.05100543051958084, -0.034589026123285294, 1.642909049987793, -0.5101089477539062, 1.3900439739227295, 1.2103828191757202, -0.3260663151741028, 2.6671183109283447, -0.10876362770795822, -0.8027604818344116, 1.0832964181900024, -0.24986036121845245, 0.7562732100486755, -0.13955754041671753, -1.553619146347046, 0.759016215801239, -0.5405297875404358, 0.651075541973114, -0.20479990541934967, -0.6427812576293945, 0.03975893557071686, 0.9559571146965027, 0.5527997612953186, -0.24516801536083221, -2.525355100631714, 0.7946208119392395, 0.08896427601575851, 0.8554133176803589, -0.33507707715034485, -0.07507558912038803, -0.3121242821216583, 0.26332008838653564, -0.08722271025180817, -0.4093080461025238, 1.8730136156082153, -0.7588939070701599, 0.7947620749473572, 0.4831961691379547, -1.281440019607544, 1.9183346033096313, -0.2124776691198349, -0.5433036088943481, 0.6034847497940063, 0.09439679980278015, -0.2875528335571289, -0.26370111107826233, -0.34035196900367737, -2.0223143100738525, 1.5351260900497437, 1.0954921245574951, -0.23191797733306885, -0.9212136268615723, 0.34487324953079224, 1.084376335144043, 0.9405297040939331, 0.486558198928833, -0.10963659733533859, 0.20566895604133606, 0.2526969015598297, 0.05455775558948517, 0.5555232763290405, 0.7086456418037415, 1.2119545936584473, 1.0390383005142212, -0.1567617505788803, 0.10046829283237457, -0.4904486835002899, 0.4502304792404175, -0.8104223012924194, 1.679840326309204, -1.1647846698760986, 0.39639660716056824, 0.44976431131362915, 0.5868809223175049, 2.9891903400421143, -0.3453468680381775, -0.39630889892578125, 0.507346510887146, -0.8636246919631958, 0.3899727165699005, 2.788048028945923, 0.08329339325428009, 0.2568548321723938, 0.43716174364089966, -0.05340449512004852, -0.7735477685928345, -1.3057154417037964, 1.8039861917495728, -0.21965591609477997, 0.3362087607383728, -0.9244131445884705, -0.626593828201294, 0.5715702772140503, -0.4028361737728119, 0.9135953783988953, 0.8058263063430786, -0.8755656480789185, 1.4999866485595703, -0.11816059798002243, 0.9673881530761719, -1.4313210248947144, -0.12003026157617569, 0.9585743546485901, -0.026693129912018776, -0.04468448460102081, -0.06471893936395645, 2.1490702629089355, -0.3245409429073334, 0.6320807933807373, -0.5381097793579102, -0.383578360080719, 0.5740225315093994, 1.2868865728378296, -1.5695816278457642, 0.5107666850090027, 0.19681452214717865, 0.026298601180315018, -0.021379247307777405, -0.9622975587844849, 0.9332522749900818, -0.6319559216499329, -0.3549959063529968, -0.48573288321495056, -1.2120885848999023, -0.3851938545703888, -1.376354455947876, -0.9344764351844788, 0.40188437700271606, -0.32389092445373535, -0.20764216780662537, -0.4909537136554718, 0.5215466022491455, 0.7770124673843384, 0.02942526713013649, 0.6949523687362671, -1.836419939994812, 0.5129597783088684, 0.17022588849067688, -0.5007901191711426, 1.0383484363555908, 0.8021070957183838, -0.36663246154785156, -0.2617701292037964, 0.36271145939826965, -0.37193745374679565, -0.5687625408172607, -1.6746898889541626, 0.8730651140213013, -0.7516005635261536, -0.5054689645767212, 0.37985438108444214, -0.24499565362930298, 0.9657680988311768, -0.0979146808385849, -1.8673925399780273, 0.9601231217384338, 0.25423818826675415, -0.01546180434525013, 0.2834683358669281, -0.2186422348022461, 0.859622597694397, 0.661320686340332, 0.40580013394355774, 0.49477219581604004, -1.523177146911621, 0.4315924048423767, -2.383768081665039, 0.3844493627548218, 1.3804408311843872, 2.0462160110473633, -0.06923765689134598, -0.2055208683013916, -1.342581868171692, 1.624799132347107, -1.6947627067565918, -0.5952433943748474, -0.32226794958114624, -0.7775636911392212, 0.36694324016571045, 0.3450537919998169, 0.135455921292305, 1.3521013259887695, 0.6694877743721008, -0.49809175729751587, -0.7169726490974426, 2.1718616485595703, -1.3404685258865356, -0.3798496723175049, 0.0854981392621994, -1.4832100868225098, 0.5055801868438721, -0.8671334385871887, 1.971063494682312, -0.3715318441390991, 0.7691224813461304, 0.3326915204524994, -0.08465104550123215, -0.43265092372894287, -0.4707046151161194, 1.0865046977996826, 0.7793607711791992, -0.25108757615089417, 0.23296865820884705, -0.7603147625923157, -0.1709664911031723, -0.6086498498916626, -0.47891849279403687, 2.356762409210205, 0.23733220994472504, -1.5883610248565674, 2.107896566390991, -0.44715678691864014, 0.9650080800056458, -0.6561185121536255, 0.039395641535520554, -0.6036797165870667, 0.061589550226926804, 1.4278604984283447, 1.6722804307937622, 2.0452699661254883, -0.3058551549911499, 0.47801679372787476, -0.32433322072029114, -0.15547962486743927, 1.8352606296539307, 0.5427495241165161, -2.321866750717163, -0.12672452628612518, 0.5074414014816284, 0.7670865654945374, -0.47581201791763306, -0.9271156191825867, -1.7602956295013428, 0.8293377161026001, 0.15931633114814758, -1.2283183336257935, 0.7925553321838379, 1.50795578956604, -0.6152002811431885, 1.2835344076156616, -1.1445600986480713, 0.45009085536003113, -0.19814561307430267, 1.0277135372161865, 0.13675495982170105], [1.5419833660125732, 0.03329082950949669, -1.2631491422653198, 0.344074010848999, 0.09253962337970734, 1.0167542695999146, -0.9760038256645203, 0.9548048377037048, -1.0269368886947632, -2.0112075805664062, -0.45075663924217224, 0.05553658306598663, 1.8923943042755127, 0.11567860841751099, -0.797221839427948, 1.4080222845077515, 1.518416404724121, 1.249260663986206, -1.4187119007110596, -0.4773699939250946, 0.9541443586349487, -1.6672111749649048, 0.055854447185993195, 0.04199981689453125, 0.013737411238253117, 0.05825261399149895, 0.3736379146575928, -0.42374491691589355, 1.378548502922058, 0.14193366467952728, 0.7754107117652893, 1.3100290298461914, 1.392555594444275, 0.7724257111549377, 1.7456796169281006, -0.17029798030853271, 0.2378583401441574, -0.07885283976793289, -0.3677898943424225, 0.440060555934906, -0.18821890652179718, 0.15602940320968628, -0.2689659297466278, -1.178993582725525, 0.831200897693634, 1.0348224639892578, 0.3682132661342621, 0.18349507451057434, 0.40185245871543884, 0.1452648788690567, 1.0599623918533325, -0.5450511574745178, -0.11202820390462875, -0.7347216606140137, -1.2554198503494263, -0.6720921397209167, 0.3375609815120697, -0.45568767189979553, -1.4453189373016357, -0.1094651147723198, 0.7909508943557739, 1.6267569065093994, 0.9893428683280945, -0.8020105361938477, 1.0344520807266235, -2.1381866931915283, 0.2458416372537613, 0.7082198262214661, 1.1504642963409424, 0.22312186658382416, -0.797712504863739, 1.1496119499206543, 1.2565094232559204, 0.27656763792037964, -1.5847113132476807, -0.6740804314613342, -0.5137516856193542, -1.3084962368011475, -1.181649088859558, -1.038490891456604, -0.11008056998252869, 0.22820107638835907, 0.9073446393013, 0.27398207783699036, 1.5481576919555664, 1.4335856437683105, -0.2821529507637024, 0.48400819301605225, 0.46994835138320923, -0.034830573946237564, 0.2641181945800781, 0.5649716854095459, -0.29572510719299316, 0.03963874652981758, 0.4375014901161194, -0.49384790658950806, 1.44437575340271, -1.3460843563079834, 1.084398865699768, -0.2669949531555176, -1.9248790740966797, -0.3103373944759369, 0.0360223613679409, -1.3134183883666992, -0.2372247725725174, 0.14984889328479767, -0.8602626919746399, 0.6166486144065857, 0.6099576950073242, -0.633034884929657, -0.4745877981185913, 0.767430305480957, 0.07653787732124329, 0.5999372005462646, -0.5921158790588379, -0.6627402901649475, 0.7002205848693848, 0.1972624510526657, 0.5025166273117065, 1.685814380645752, 0.4048061966896057, -0.1303189992904663, -1.6746212244033813, 0.27698540687561035, 0.5200858116149902, 0.22143813967704773, 1.1771094799041748, 1.5865280628204346, 0.21322549879550934, -0.6809890270233154, -0.45849207043647766, 0.020999249070882797, 1.3756108283996582, 0.16729792952537537, 1.320385217666626, -1.1234748363494873, 1.2470191717147827, 0.6521049737930298, -0.23082496225833893, 0.4470270574092865, 1.1180700063705444, 0.6743082404136658, 1.1417038440704346, 0.6131512522697449, 0.8310059905052185, 1.0453847646713257, 0.1318836659193039, 0.1500466763973236, 0.7562556862831116, 0.7700712084770203, -1.0577754974365234, -0.6119639277458191, -1.1457581520080566, -0.22200323641300201, -0.8088290095329285, 1.37744140625, -1.4322701692581177, -0.27533024549484253, 0.3538517355918884, -1.7304182052612305, 0.9343510866165161, -0.22732006013393402, -0.037871427834033966, 0.5701348781585693, 2.2180540561676025, -0.11502029746770859, -0.47705045342445374, 0.6452286243438721, -1.2337216138839722, 0.5074419975280762, 0.5249574184417725, -1.678446888923645, -0.4883384108543396, 1.575673222541809, 0.7507332563400269, -0.03053421340882778, 1.9606841802597046, -2.074122667312622, -0.06664688140153885, 0.6264083385467529, 0.44504743814468384, 0.5134199857711792, -0.7018601894378662, -0.8498713970184326, 1.9003005027770996, 0.40764179825782776, 0.8183737993240356, -0.7729602456092834, -0.8557814955711365, -0.3554303050041199, 0.9730470180511475, 0.5771521329879761, -0.5115997791290283, -0.5322303771972656, 1.5490999221801758, 0.19931501150131226, 0.4498286843299866, 0.9734077453613281, 1.5507402420043945, -0.22585132718086243, 0.7700297832489014, -0.43982335925102234, 2.0987257957458496, 0.43209967017173767, 0.09472857415676117, 0.2846527695655823, -0.6286530494689941, 0.6450279951095581, 0.0830150917172432, 0.38494327664375305, 0.5659579634666443, -0.2599296271800995, 0.19029048085212708, 1.5583231449127197, 1.9515631198883057, -0.25146129727363586, 0.5221169590950012, -0.312272846698761, 0.3665103018283844, 2.0818166732788086, -1.7254995107650757, -0.871235191822052, 0.9971799850463867, 1.6909842491149902, 0.32263240218162537, -1.3353466987609863, -0.7073681354522705, -0.3008096218109131, 1.571364402770996, 1.1946922540664673, 0.6872475743293762, 3.1719555854797363, -1.646079659461975, 0.27750515937805176, -0.11646474897861481, 1.074398398399353, 0.4323534667491913, -0.45388782024383545, 1.2770843505859375, 0.12034802883863449, -0.4221932291984558, -0.19321979582309723, -1.2799134254455566, 1.0438913106918335, -1.6885536909103394, -0.07683508098125458, 0.8865239024162292, 0.9025394320487976, 0.8326643705368042, 0.40137141942977905, 0.32493162155151367, -1.249244213104248, -0.6594018936157227, -1.4879403114318848, 1.2527153491973877, 0.21113267540931702, 1.5097121000289917, 1.3112448453903198, -0.12459177523851395, -0.29125261306762695, 0.15481680631637573, -0.39323464035987854, 0.45520350337028503, 0.9310211539268494, -0.26385298371315, 1.7679697275161743, -0.762860119342804, -1.1044753789901733, -0.48534300923347473, 0.634936511516571, 0.3864682912826538, -1.0909335613250732, -0.6068747639656067, 1.2402552366256714, 0.5230608582496643, -2.166438102722168, 0.27121421694755554, -0.17498956620693207, 0.14076340198516846, 0.9339569807052612, 1.1140490770339966, 0.4587056338787079, 1.213120937347412, -0.7705376148223877, -0.2677188217639923, 1.113088846206665, -1.673121690750122, 0.14334037899971008, -0.5178122520446777, 0.15837620198726654, -0.03127862885594368, -0.7076389193534851, 0.6308673024177551, -1.2259873151779175, 0.4234137237071991, -0.06235792487859726, -1.3647334575653076, 0.6492087244987488, -0.3041384816169739, 1.0569771528244019, -0.14371201395988464, 0.32022103667259216, 2.201437473297119, -0.0011069821193814278, 0.5571980476379395, 0.7061403393745422, 1.2240092754364014, -0.09437218308448792, -0.09104501456022263, -1.2604745626449585, -0.3134593963623047, -0.4796338379383087, -0.5970747470855713, 0.03774450346827507, 0.6265586018562317, -1.0146541595458984, 1.6636074781417847, 1.305154800415039, 0.3128124475479126, 0.9667807817459106, -1.4836061000823975, -0.301615446805954, -0.07992472499608994, 0.1706649214029312, -0.6357991695404053, 0.07454049587249756, -0.008642049506306648, 1.68630850315094, 0.7228384613990784, -0.04646238312125206, -0.18475459516048431, -0.4743667542934418, 0.07757454365491867, -0.8210821151733398, 0.23187072575092316, 1.7113744020462036, -0.9180041551589966, -2.014450788497925, 1.3465343713760376, -1.1249604225158691, -1.6723394393920898, 0.11680914461612701, -1.0844916105270386, -0.7879948019981384, 0.52069491147995, 0.5671910047531128, 0.38080349564552307, -0.21461006999015808, 0.8695266842842102, 0.43844926357269287, -0.26774147152900696, 1.5032988786697388, -0.2776227593421936, -0.28794264793395996, -0.3020627498626709, 0.15825685858726501, -0.2251136153936386, 0.030152857303619385, -2.6855227947235107, 1.2132396697998047, 0.30714190006256104, 0.744663417339325, -0.7386519312858582, 2.282026767730713, 1.8621960878372192, -1.221285104751587, -0.8150803446769714, -1.6550811529159546, 1.033289909362793, 0.7074235677719116, -0.18007950484752655, 2.45365571975708, -2.3328590393066406, -2.0196080207824707, 1.0254658460617065, 1.4862719774246216, 0.7180656790733337, 1.0522032976150513, 1.0526334047317505, -2.0616137981414795, -0.9282088279724121, -0.6845653653144836, -0.807471752166748, -1.1412076950073242, 0.8644338250160217, 0.17701151967048645, -0.3213983178138733, -1.313594937324524, 0.05538565665483475, 0.5521374344825745, 0.7033114433288574, 1.0724271535873413, 0.5459430813789368, -0.3302827477455139, 0.43206191062927246, -1.4545888900756836, -0.3511306941509247, 1.4028784036636353, 0.29296550154685974, 1.5197256803512573, -0.007910938002169132, -0.6993071436882019, 2.9608891010284424, -0.6653543710708618, 0.15363679826259613, 2.367499589920044, -0.7124156355857849, -1.564648151397705, -0.3651380240917206, 0.8683984279632568, -1.0779801607131958, -1.4984405040740967, -0.17910946905612946, -0.6429880261421204, 0.18525172770023346, 0.3602816164493561, -1.5843188762664795, 1.3146674633026123, -0.5921146869659424, -0.6175894141197205, 0.7943690419197083, 0.12965300679206848, 0.4561786949634552, 0.781883716583252, 0.49988359212875366, 0.24383606016635895, -1.2283368110656738, -0.8607149720191956, -0.5142956376075745, 0.4772038757801056, 0.6820409297943115, 0.4670776128768921, 0.5385591387748718, -0.9665507078170776, -0.48943397402763367, -1.3014566898345947, -0.2621902823448181, -1.454444408416748, -0.46239960193634033, -2.1152312755584717, -0.6192602515220642, -0.41031602025032043, 1.1647032499313354, -0.5226447582244873, 0.48319175839424133, 1.4403891563415527, 1.4172652959823608, 0.4862161874771118, -1.6357319355010986, -0.43220242857933044, 0.8642532825469971, 0.03810169920325279, 0.6876357793807983, -1.0522730350494385, -0.370033860206604, 0.41742342710494995, -1.4410334825515747, 0.2943457067012787, 0.20873069763183594, -0.9846866726875305, 0.7974342703819275, 1.374219298362732, 1.052905559539795, 0.10760953277349472, -0.05393601581454277, 0.3310606777667999, 1.2232416868209839, -0.8754714727401733, 1.406256914138794, -0.21156705915927887, -0.23777155578136444, -0.3528509736061096, 2.098296880722046, -0.724128782749176, 0.9670973420143127, 0.18824365735054016, -0.8238368034362793, -0.5542449951171875, 1.2405563592910767, 0.5639222264289856, -0.1198815330862999, -0.19221188127994537, -0.44248706102371216, 1.1620782613754272, 1.2263851165771484, -1.5188884735107422, 0.2739797532558441, 0.4407263994216919, -1.1116431951522827, -0.572824239730835, -0.23746643960475922, 0.40962475538253784, -0.21150514483451843, -0.3001190721988678, -1.1000005006790161, 0.6853576302528381, -0.038948915898799896, -0.13824652135372162, -0.2924332022666931, -1.0820438861846924, -1.28849196434021, 0.6609392166137695, -0.9237057566642761, -0.6787837147712708, -1.3648922443389893, 0.6923794746398926, 0.7586212754249573, 0.8986443877220154, -0.598563015460968, 0.175231471657753, 1.145768165588379, -0.9840343594551086, -0.9497076272964478, 1.1901081800460815, -0.8419787883758545, -2.332427740097046, -1.5351898670196533, -1.3772436380386353, -0.7532730102539062, -0.3289835751056671, 0.30259934067726135, -0.920737624168396, 0.13436675071716309, 1.06133234500885, 0.3807411789894104, 3.56795072555542, 1.3336410522460938, -0.0629827231168747, 0.18392322957515717, -0.5850589275360107, 0.05539698153734207, 0.12413384020328522, -0.16939441859722137, 0.46009400486946106, 1.1972423791885376, 0.09042444080114365, -0.9146829843521118, 0.6871934533119202, 0.572272777557373, 2.2127630710601807, -0.14643393456935883, -0.6186737418174744, 0.5244547724723816, -0.30424413084983826, 0.4304141700267792, 1.1028499603271484, 0.8142206072807312, 1.2329070568084717, 1.0589667558670044, -0.7123379707336426, 0.16807222366333008, -0.33598390221595764, 0.21161434054374695, -0.9054358005523682, 0.0611727274954319, -0.19992004334926605, 0.6704902052879333, -1.1615660190582275, -1.383392572402954, -1.0069384574890137, 0.009429015219211578, -0.5601580739021301, -0.7176916003227234, -0.8949154019355774, 1.691205620765686, -1.1715891361236572, -0.4636026620864868, 1.3886442184448242, -0.24050793051719666, -0.8866854310035706, -0.7773311734199524, -0.9875670075416565, 1.4897619485855103, 1.4184918403625488, 0.3282889723777771, 1.7440794706344604, -0.41498491168022156, 0.3410469889640808, 0.6407409906387329, 0.3219364881515503, -0.012612254358828068, -0.9388310313224792, 0.38674744963645935, -0.17498895525932312, 0.5438105463981628, -2.2721762657165527, 0.36195576190948486, -0.2933202087879181, 0.34699365496635437, -1.0106279850006104, 0.883223295211792, 1.1741821765899658, -1.0216264724731445, -2.3130147457122803, -1.5588414669036865, -0.5646703243255615, -0.03794748708605766, 0.9644319415092468, 1.3001668453216553, 0.17052499949932098, -1.0348589420318604, 0.3556004762649536, -0.834872305393219, -1.646485447883606, 0.9472529888153076, 0.23379051685333252, -0.23919646441936493, 0.9747881889343262, -0.4694947898387909, 0.4373168647289276, -0.5223269462585449, -0.8887767791748047, 0.951626718044281, -0.7254688739776611, -1.3334901332855225, 0.9523716568946838, 0.5402659773826599, 1.0461560487747192, -1.9050681591033936, 0.29636624455451965, -1.2944202423095703, -0.40244773030281067, -0.26997101306915283, -0.8096369504928589, -0.8026348352432251, -1.338985800743103, -0.2305590659379959, -1.9215677976608276, -2.229032039642334, 0.5666467547416687, 0.9450455904006958, 0.0996093675494194, -0.5653381943702698, -1.0297372341156006, -0.6826286911964417, -1.9779504537582397, 0.09357182681560516, 0.9249580502510071, 1.8011976480484009, 0.3225092887878418, -1.136582851409912, 1.6316698789596558, 1.2258753776550293, 0.5363245010375977, -0.9660948514938354, 0.2496425360441208, -1.1571065187454224, 0.15935957431793213, 0.9544628262519836, -0.4429158866405487, 0.6198605895042419, 0.33259448409080505, -0.13926397264003754, -1.1828991174697876, -1.0023802518844604, -0.6371804475784302, -2.086562395095825, 0.01314813457429409, -0.6543765068054199, 0.2513366937637329, 0.35078948736190796, -0.025717519223690033, 0.786629855632782, -0.9748731851577759, -1.4094657897949219, 0.22761139273643494, 0.2770756185054779, 0.6311420798301697, 0.3472185730934143, 0.32268133759498596, -0.8291063904762268, -1.2927136421203613, 0.16287024319171906, -1.998913288116455, -0.03388554975390434, -0.708509087562561, -2.1368215084075928, -1.3251891136169434, -1.709580898284912, 2.0888783931732178, -1.4796557426452637, 0.3249761760234833, 0.40084972977638245, -0.5103784799575806, -0.3923795223236084, 0.37624579668045044, 1.916684865951538, 1.0203217267990112, -0.6310632228851318, -1.4188337326049805, 0.6383673548698425, 0.6994829773902893, -1.2520596981048584, -1.7508773803710938, 0.23808439075946808, -1.6417045593261719, 0.06369362026453018, -2.8638784885406494, -0.5121569037437439, -1.5759854316711426, 0.9911691546440125, 1.61319899559021, 0.16826224327087402, 1.0728284120559692, 0.06641809642314911, 0.5722687244415283, 1.806087851524353, -0.5502312779426575, -1.8213324546813965, -0.005148874595761299, 2.283489942550659, 1.2661100625991821, 2.0418550968170166, 0.06255951523780823, -0.31626468896865845, -1.307456135749817, -0.5933854579925537, -0.4379447102546692, -0.39421552419662476, 0.8066501617431641, 0.4053013026714325, 1.3750487565994263, -0.502264678478241, -0.5367162227630615, -0.7482833862304688, 1.0612506866455078, -0.1956312358379364, 0.4204866290092468, 0.13539977371692657, -0.4798395335674286, 0.5814936757087708, 1.3475449085235596, -0.4896393120288849, -1.8439393043518066, 0.9699366688728333, 0.8664035797119141, -0.01889997348189354, 0.9574966430664062, 1.2873728275299072, 1.3165193796157837, 1.5543967485427856, -0.31253302097320557, 0.8773953914642334, -0.8592864274978638, 0.06479943543672562, 1.1706972122192383, 0.2095934897661209, -0.38123857975006104, 0.8329847455024719, 0.8523194193840027, 1.0877456665039062, -2.706838607788086, -1.1539233922958374, 0.3441007435321808, 1.1028186082839966, 0.6258363127708435, -1.1368134021759033, -0.28303438425064087, -0.2807702124118805, -1.7007853984832764, -1.2596533298492432, -0.8495974540710449, -1.2257511615753174, 1.6882396936416626, -0.6997112035751343, 0.1457880735397339, -0.8574202656745911, 0.2530701458454132, -1.2436518669128418, -1.2743386030197144, 1.643984079360962, -0.19217921793460846, -1.1682243347167969, -1.395063042640686, -0.7591588497161865, 0.964364230632782, 0.8291627168655396, 0.9310388565063477, -0.2982458174228668, 0.3286254107952118, 0.20639188587665558, -0.6174749732017517, -0.5954110026359558, 0.2206779420375824, 0.9688808917999268, 1.4513859748840332, -0.9950783848762512, -0.2037268877029419, 0.12189039587974548, 1.601920485496521, -1.1854873895645142, -0.3344455063343048, -1.6301119327545166, -0.934933602809906, -1.1034038066864014, -1.6203292608261108, -0.9925321936607361, 0.5689323544502258, 0.13575169444084167, 2.1829516887664795, -0.15210992097854614, 1.03878915309906, -2.2973177433013916, -1.9982426166534424, 1.1311006546020508, 0.9827473759651184, -0.30308133363723755, -0.8849696516990662, 1.1592388153076172, -0.44191035628318787, -0.3660805821418762, -0.9068123698234558, -0.012338340282440186, 0.7442436814308167, 1.2778009176254272, -0.5288992524147034, 0.1965782195329666, -0.5172207355499268, 1.994565725326538, -1.4643940925598145, -1.6899020671844482, 1.0244262218475342, 0.7385838627815247, 0.6801588535308838, -0.9018722176551819, -0.4713539481163025, 0.501465916633606, -0.45429328083992004, -0.6258816719055176, -0.5558122396469116, 1.8632776737213135, -2.372843027114868, 0.21843993663787842, -0.7540668249130249, -0.790404200553894, 0.30264294147491455, 0.86191326379776, -0.4605056941509247, 2.746856212615967, -2.035175323486328, 0.2844705581665039, 0.5304864645004272, -0.5944528579711914, -0.23502089083194733, -0.5209451913833618, -0.230324849486351, 0.020290235057473183, 0.15869081020355225, -2.5676369667053223, -0.033588092774152756, -0.16466861963272095, -2.6831371784210205, 0.7808482050895691, -0.03698981925845146, 0.8600031733512878, -1.082044243812561, 0.9882801175117493, 0.23126034438610077, 2.1827478408813477, -3.2007222175598145, -1.1996794939041138, -2.9606127738952637, 1.5272208452224731, 0.3435983955860138, 2.1198196411132812, 0.8987952470779419, -0.6163374185562134, 1.6662238836288452, -0.39394882321357727, 0.145126074552536, 2.468116521835327, -0.7738761305809021, 1.0213109254837036, -0.24924099445343018, 0.7684609293937683, 0.19167469441890717, -0.1307147592306137, 1.2904739379882812, 1.498754620552063, -2.0168650150299072, -1.8344247341156006, -0.9334522485733032, 0.7591123580932617, -0.13658258318901062, -1.5346102714538574, 0.34093788266181946, -0.5404829978942871, -0.9504693150520325, -1.224109411239624, -0.2843164801597595, -0.380435585975647, 1.1766571998596191, -1.2726694345474243, -0.13349714875221252, 0.22774624824523926, -0.6215961575508118, 0.4295446574687958, 1.2298095226287842, 0.13670776784420013, -0.2004203051328659, -1.2978317737579346, -0.6769388914108276, -0.11681350320577621, 0.9851891398429871, 0.09830936044454575, -0.62091463804245, 0.13633063435554504, 0.5572341680526733, -0.09827329963445663, -0.4546144902706146, 0.2777976989746094, 0.26668527722358704, -2.0613112449645996, -0.9946417808532715, -1.6116691827774048, 1.7759344577789307, 0.008884531445801258, -0.3736242949962616, -0.462209552526474, -0.15494653582572937, 0.12185267359018326, -1.1171449422836304, 0.6538978219032288, -0.8923137187957764, -1.434787392616272, 2.599130630493164, -1.272359013557434, 0.4020821452140808, 0.18039163947105408, 0.1798269897699356, -0.6347090601921082, -0.22542616724967957, 0.9579179883003235, -0.537982702255249, -0.2019181251525879, 0.23037412762641907, -0.07082907855510712, -1.2485196590423584, 1.4367648363113403, 1.1129387617111206, 0.5287482142448425, -0.14610058069229126, -0.4545418918132782, 0.014711618423461914, -1.450918436050415, -0.9413972496986389, -0.3835086226463318, -0.1110559031367302, -0.16488954424858093, -0.6541319489479065, 0.4968682825565338, -0.4607379138469696, -0.3831747770309448, 2.3226373195648193, -1.3282487392425537, 0.044105466455221176, 2.435445547103882, -0.7553452849388123, -1.655365228652954, -1.437441110610962, 0.38248005509376526, -0.27495166659355164, 1.2271735668182373, 0.405990868806839, 0.7370147109031677, -0.2732369899749756, -0.1198788657784462, 0.34457799792289734, 0.4149656593799591, -0.9803626537322998, -0.8078548908233643, 2.009554862976074, 0.16533935070037842, -2.323991060256958, -0.7841596603393555, 0.3969343602657318, -0.6608436107635498, 1.2091110944747925, -1.9079455137252808, 0.226007878780365, -0.704684853553772, 1.3896024227142334, 1.0786268711090088, 0.6440380811691284, 0.36628103256225586, 1.6437699794769287, -2.0882275104522705, 0.15291716158390045, 0.5144861936569214, 1.8219090700149536, 0.36002084612846375, 1.0763949155807495, 0.6395936012268066, 0.4389919340610504, -0.58163982629776, 0.8902316689491272, -1.585006833076477, 0.10813190788030624, -0.3133467435836792, -2.1370630264282227], [0.056707996875047684, 1.1718637943267822, 0.7322306632995605, 0.5492442846298218, -0.02430884726345539, -0.7173755168914795, -0.5659865140914917, -2.002044200897217, 1.4712169170379639, 0.804969072341919, -0.4911953806877136, 0.10170886665582657, -1.5503100156784058, 1.0726628303527832, -0.6829448342323303, -0.3320455849170685, -2.253458023071289, -1.6758456230163574, 0.1499268263578415, -0.4949462413787842, 0.3911455571651459, 0.5201542377471924, -0.2042209953069687, 0.09780041873455048, 1.343245506286621, 1.9812061786651611, 0.16905754804611206, 0.6206749677658081, -0.21901936829090118, -1.7497279644012451, 0.47258836030960083, -1.066673755645752, -1.3328694105148315, -0.792037308216095, 0.29211440682411194, 1.020511507987976, -0.08469045162200928, -1.03555428981781, -0.8381768465042114, 0.21179993450641632, -0.1882709413766861, -0.2021469622850418, -0.21977613866329193, -0.8156774640083313, -0.6791977882385254, -0.9771034717559814, 0.1900426149368286, -0.6608357429504395, 1.0433712005615234, 1.7966867685317993, -0.13941611349582672, 0.3399154543876648, 0.5650801062583923, -0.6933538317680359, -0.9059010148048401, 0.8732523322105408, 0.6371913552284241, 1.00072181224823, -0.18091754615306854, 0.3825458586215973, -0.6263382434844971, -1.2482578754425049, 0.40811336040496826, -1.526369571685791, -0.31966632604599, 1.0067741870880127, -0.657130777835846, 1.6862503290176392, -1.643067717552185, -0.26368463039398193, 1.0236570835113525, 0.7911830544471741, 0.24018742144107819, -0.026125626638531685, -0.22288423776626587, 0.023537233471870422, -1.6826965808868408, 0.01905490644276142, -1.517408013343811, 0.3547455966472626, -0.2940763533115387, -1.7362257242202759, 2.5194509029388428, 0.25066617131233215, -0.4903227984905243, 0.5333260297775269, 0.5514765977859497, 0.3260079622268677, 1.1533957719802856, 0.17204663157463074, -0.8401803970336914, 1.2029796838760376, -2.112583875656128, 0.028101423755288124, -0.213910311460495, -1.1777863502502441, -0.6053476333618164, -0.16433387994766235, 1.8149083852767944, 0.5387120842933655, -0.5649818181991577, -1.274820327758789, -1.2932345867156982, 0.6618387699127197, 1.213690161705017, -0.4877850115299225, -0.0036227311939001083, 0.9573068022727966, 0.9775368571281433, 0.8590018153190613, 0.19726839661598206, -0.4337151050567627, -0.17449833452701569, -0.3872487246990204, -1.1181426048278809, 0.8419092893600464, 1.968459129333496, 0.6181021332740784, 0.5616270899772644, 0.9401994347572327, -0.014009521342813969, -0.7574827671051025, 0.6141722202301025, 0.4097226858139038, -1.4108316898345947, 1.049096703529358, -0.7918463349342346, 0.9244181513786316, 0.11809256672859192, -0.5093831419944763, -0.9023868441581726, -0.8913450241088867, -1.903050422668457, -0.9664616584777832, 0.14316031336784363, -1.3665956258773804, -0.42221447825431824, 0.4133603274822235, -2.4276695251464844, -1.1392827033996582, 1.040285587310791, -0.6485775113105774, -0.38399040699005127, 0.8668878078460693, -1.1804097890853882, -2.6991631984710693, 0.6362320780754089, 0.27453678846359253, -1.0568989515304565, -2.4216535091400146, 1.4366099834442139, -0.1294407695531845, 0.0497627928853035, 0.6023318767547607, 1.172268271446228, -0.4232030510902405, -0.0400414876639843, 1.7341089248657227, -0.05673541501164436, 0.06340647488832474, 0.9496986865997314, -2.5355048179626465, 0.2659797966480255, -0.9683730006217957, 0.08427606523036957, -0.29010921716690063, -0.48609036207199097, 0.6778541207313538, -0.531997561454773, -2.32395601272583, 1.6361981630325317, -0.6494061350822449, -0.23916758596897125, 0.9588046073913574, 0.5220412015914917, -0.5864295363426208, -0.3439117670059204, 0.6989185214042664, -0.5962659120559692, 0.49474406242370605, 2.6163265705108643, -0.370500773191452, -2.428194284439087, -1.3609024286270142, 0.6181468367576599, -1.274989128112793, 0.5614119172096252, 0.6585406064987183, 1.5771280527114868, -1.0982526540756226, 1.7431774139404297, -1.529363751411438, 0.6275295615196228, 1.2844804525375366, 0.8070903420448303, 0.3111852705478668, 0.24605663120746613, -0.24233746528625488, -0.16226300597190857, 0.5288872122764587, 0.5798529386520386, 0.1177249625325203, -0.34869253635406494, -0.5765216946601868, 0.005596677772700787, 0.3794407844543457, 0.9692389965057373, -0.853350818157196, 0.028652789071202278, -1.0704807043075562, 0.9942770600318909, 0.3687689006328583, -0.976706862449646, -1.9047307968139648, 0.9027083516120911, -0.9113925099372864, -0.6410226821899414, 1.7332744598388672, -0.5514804720878601, -0.4371192455291748, 1.036794662475586, 0.2989693284034729, 1.1333638429641724, -0.6532132029533386, -0.6253604888916016, -0.7654910087585449, 1.8821603059768677, -0.8932624459266663, -0.4585098922252655, 0.008837336674332619, 2.499718189239502, 0.2828422486782074, 0.24116986989974976, -1.7630915641784668, 0.3010311722755432, -0.43469613790512085, -0.7055643796920776, -0.26415327191352844, 1.8949826955795288, -1.102344036102295, 0.09735990315675735, 1.2725530862808228, -0.703076183795929, -1.9225471019744873, 0.6378545761108398, -0.4862716495990753, -0.7299171686172485, 1.5057034492492676, 0.07961864024400711, 1.1416279077529907, -0.428569495677948, -0.9316014647483826, 1.597256064414978, 1.4928200244903564, -1.0261499881744385, -0.8135435581207275, 0.3844607174396515, -0.6162068843841553, 1.3017735481262207, -0.8635827898979187, -0.9880411028862, -1.772209644317627, -0.5827480554580688, -0.8257375955581665, 0.013358662836253643, 0.8009006381034851, -0.08026918023824692, 2.3974599838256836, -0.8156939148902893, -0.442345529794693, 0.3106784224510193, 0.24211162328720093, 1.5151864290237427, 1.0618388652801514, 1.2466981410980225, -0.3765254318714142, 0.5388237833976746, 0.402908056974411, -0.17082034051418304, -1.3185019493103027, 0.05728363245725632, 1.0532276630401611, -0.7531765699386597, -0.17403605580329895, 0.2536143362522125, -0.3392748236656189, 0.8563727736473083, 1.4818000793457031, -0.6996344327926636, -0.6509300470352173, -2.4715373516082764, -1.005864143371582, -0.02034689672291279, -0.8986450433731079, 0.9400944709777832, -0.5382935404777527, -0.42435401678085327, 0.05300808697938919, 1.5722119808197021, 1.1752506494522095, 0.08627625554800034, 0.027432719245553017, 1.0176563262939453, 0.9600756764411926, 0.022916346788406372, 0.2118155062198639, -1.2169520854949951, 0.18749584257602692, -0.8038721084594727, 1.864759922027588, -0.08729139715433121, 0.3213406801223755, 0.6381263732910156, 1.542862057685852, -1.0612622499465942, 1.194148302078247, -1.6977143287658691, 0.36160531640052795, 0.8361576795578003, -1.4900370836257935, -0.24896417558193207, -1.3913226127624512, -0.6614759564399719, 0.2933118939399719, 0.8901253342628479, -0.4770892262458801, -0.1438378542661667, -1.1844538450241089, 0.5382644534111023, -0.4662860929965973, 0.10066282749176025, 0.5513992309570312, -1.6709558963775635, -0.7584213018417358, -1.4042857885360718, 1.2144373655319214, 1.0104904174804688, -2.0841267108917236, 0.7231981754302979, 1.2066450119018555, -0.27090564370155334, -2.70102858543396, 1.0958466529846191, 0.8746693730354309, -0.5805957913398743, -0.7620220184326172, 1.2109323740005493, 0.20585905015468597, -0.19695641100406647, 0.2157033532857895, -1.8431984186172485, 1.1777973175048828, 0.6944831013679504, 0.6685329675674438, -0.7472633123397827, -0.5205204486846924, 0.08632581681013107, -0.3022903800010681, 0.7810683846473694, 0.05616603419184685, -1.0604732036590576, -0.8047637343406677, 0.23876214027404785, -0.704817533493042, -2.1479039192199707, 0.437939316034317, 0.6097338199615479, -1.9104583263397217, 0.3654819428920746, -0.9931632280349731, -1.325021505355835, 0.6587367057800293, -0.693535327911377, 0.2627487778663635, -1.0312682390213013, -0.1892162710428238, 0.3099629580974579, 1.167224407196045, -0.6292112469673157, 2.372995376586914, 0.3607313930988312, 0.7044907212257385, 1.4435501098632812, 1.5431404113769531, 0.12888562679290771, 0.6275327205657959, 0.21398061513900757, -0.022275026887655258, -0.32372787594795227, -0.6497630476951599, -1.448175072669983, -0.5592217445373535, -0.3353319466114044, 0.5391497015953064, -0.24169112741947174, 1.1231350898742676, 0.7698197960853577, -0.9826779961585999, -0.6667818427085876, 0.7018977403640747, -1.3177913427352905, -0.6441076397895813, -1.0073879957199097, 1.366982340812683, -0.17378835380077362, -0.6223303079605103, 0.10943055897951126, 1.6529115438461304, -1.5274139642715454, 1.2256767749786377, 0.7323312759399414, 0.2069614678621292, -0.6268166899681091, 1.0838770866394043, -1.30106520652771, -0.4375074505805969, 0.9454474449157715, -0.7385927438735962, -0.36574387550354004, 0.9505068063735962, 1.6599632501602173, -0.6443474292755127, 0.42756348848342896, -0.1485641598701477, 0.8049112558364868, -0.431856632232666, 1.6520352363586426, 0.6202669739723206, -0.17503902316093445, 1.7637012004852295, -0.9477916955947876, -0.38769975304603577, 2.250349998474121, -1.3377636671066284, 0.2497912496328354, -0.17194636166095734, -0.3791312575340271, -2.2066330909729004, -0.9332894682884216, -1.6220088005065918, -1.7179625034332275, -2.3280189037323, -0.3419792652130127, 0.6106951832771301, 0.2112167775630951, -0.9995719194412231, -1.2879829406738281, -0.3788527250289917, 1.934456706047058, -0.22445322573184967, -0.5522451996803284, -0.02005827985703945, 0.14495475590229034, 0.21254666149616241, -1.410226583480835, -1.4181807041168213, -0.23998315632343292, 0.924734354019165, 0.25012847781181335, 0.2934882342815399, 1.7129400968551636, -1.3578754663467407, 0.4659017324447632, -0.14325891435146332, 1.083686113357544, -0.28654903173446655, -0.09095079451799393, 1.2225888967514038, 0.1432407796382904, -0.44849467277526855, -1.1524810791015625, -1.2054039239883423, 1.3700512647628784, 1.2041432857513428, -0.6585265398025513, 1.7231720685958862, 0.1343955099582672, -0.1655004620552063, 1.6127492189407349, 0.5558037757873535, -1.592374563217163, -0.3665272295475006, -0.4579731822013855, -1.513256311416626, 1.6852481365203857, -0.6779320240020752, 0.5580978989601135, -0.833055317401886, 0.6007510423660278, 1.703766107559204, -0.005395069252699614, 0.7613546848297119, 1.0616339445114136, -1.3194105625152588, 1.012704849243164, -0.2422015368938446, -1.0682545900344849, -0.09739982336759567, 1.0618221759796143, -1.3834636211395264, -0.6894869208335876, 0.4552314877510071, 0.14480991661548615, 1.8304641246795654, -1.6213791370391846, 1.5674045085906982, -1.345879316329956, 0.23592978715896606, 0.9324973225593567, 0.9762954115867615, -0.5761879086494446, -0.2908301055431366, 1.1018325090408325, 0.15340882539749146, -0.12920774519443512, 1.0670721530914307, -1.4185081720352173, -1.5507601499557495, 0.03980787470936775, -0.9643332958221436, 0.5496811270713806, 0.9156278967857361, -1.3666294813156128, -0.7791489362716675, 0.1696540117263794, -0.2372729629278183, 0.925030529499054, 0.7434568405151367, 0.2630643844604492, 0.2799729108810425, -2.137009382247925, 0.5872085094451904, 0.2755489647388458, 0.42843660712242126, 0.22810006141662598, -0.19606229662895203, -0.4392046630382538, -0.8619016408920288, 0.8747067451477051, -0.029419943690299988, -0.21813051402568817, 0.6031129956245422, -0.2800516188144684, 0.45208489894866943, -0.9866793155670166, -0.8208081722259521, -0.7000563144683838, 0.10165884345769882, -0.9101983308792114, -1.6514205932617188, -0.6486722230911255, -0.90534508228302, 1.1445016860961914, 0.025920992717146873, -0.44034484028816223, -0.189665749669075, -0.570202648639679, 1.387877106666565, -1.7432340383529663, 0.11007140576839447, -1.0773639678955078, -0.2336946278810501, 0.9129096865653992, 0.39544299244880676, 0.3559601902961731, 1.756888508796692, -0.6025413870811462, 1.1600464582443237, -0.7071794271469116, -1.0379997491836548, 0.12225554138422012, -0.000785421347245574, -1.5060704946517944, 0.8578230738639832, -0.8295950293540955, 0.16138048470020294, 0.021440068259835243, -0.48025810718536377, 0.6254069805145264, 1.7618316411972046, 1.038388729095459, 1.094940423965454, -2.3840525150299072, -0.2625579833984375, -0.8571861386299133, 0.6879544854164124, -0.13559140264987946, 0.2827167212963104, 0.9777677059173584, 1.1965123414993286, 0.06448899954557419, -1.0274357795715332, -1.2353556156158447, -0.04222206026315689, -1.523730754852295, 2.5061028003692627, 0.261909544467926, -0.46798887848854065, 1.4546983242034912, 0.5009410381317139, 1.4010034799575806, -1.763049840927124, -0.5816553235054016, -1.559038758277893, -1.7035857439041138, -1.6760718822479248, -0.1600153148174286, -1.642688512802124, -0.2743339538574219, -0.41242533922195435, 1.2957454919815063, 0.9810296893119812, -0.5079570412635803, -0.08664459735155106, 1.2019548416137695, -2.0284922122955322, 2.0053813457489014, -0.8971356153488159, -2.8104357719421387, 0.4715564250946045, -0.14827218651771545, -0.2339976280927658, 1.5088509321212769, -1.0043895244598389, 0.9197815656661987, 0.4382227957248688, 0.848770022392273, -0.3432616591453552, 0.27218955755233765, -0.10195630043745041, 0.978747546672821, -0.3065640330314636, 0.6462984085083008, -0.12809737026691437, -1.1012792587280273, 1.3538285493850708, -1.3123341798782349, -1.2593199014663696, 0.31396254897117615, 0.452392578125, 0.13160833716392517, 1.29840886592865, -0.650629997253418, 1.0621761083602905, -1.5350613594055176, 0.6714267134666443, -2.7964582443237305, 1.3104718923568726, 0.3497183620929718, -0.7670484185218811, 0.746568500995636, 1.354693055152893, 1.5068844556808472, -0.5439556241035461, 0.024448363110423088, -1.1612331867218018, 0.3160669505596161, -1.2209036350250244, -0.1258445680141449, 2.839808702468872, -0.11862420290708542, 0.45292380452156067, -0.0474034920334816, -0.04342251271009445, 1.1686733961105347, -0.7625150084495544, -0.12073778361082077, -1.5528627634048462, 0.597678542137146, -0.8088628649711609, -0.9302616715431213, -0.36238420009613037, 1.2400317192077637, 0.8411188721656799, -0.3454612195491791, -0.3978533446788788, -0.8275464177131653, 0.4456816017627716, 0.36964303255081177, -0.5689741373062134, 1.2623223066329956, 1.0107730627059937, 1.2678941488265991, -1.1696912050247192, -0.891798198223114, 0.4569692015647888, 0.22993645071983337, -0.39663365483283997, 0.7415580153465271, 0.18140342831611633, -0.26623526215553284, -0.47042325139045715, 0.7102846503257751, -2.2154414653778076, -2.3528964519500732, 1.0774216651916504, -0.6241108179092407, -1.1987875699996948, 0.4390656352043152, -0.9071283936500549, 1.3655236959457397, -0.1894955188035965, -0.20692960917949677, -0.6071268320083618, -0.4201209247112274, -0.5960983037948608, 0.47186213731765747, -0.6229886412620544, -1.6977800130844116, 0.1605379581451416, -0.6006684899330139, 0.8500820994377136, -1.1715641021728516, 0.28844153881073, 0.7321777939796448, -0.323184996843338, -1.312855839729309, -0.9112293124198914, 1.367443561553955, 2.555400848388672, -0.16830086708068848, -1.7447798252105713, -0.06331715732812881, -1.2242716550827026, 0.5198675394058228, 0.5558537244796753, 0.21567551791667938, 0.6654741764068604, 0.10518667846918106, 1.3635534048080444, -1.8858908414840698, -0.28102827072143555, 1.24319326877594, 0.10196707397699356, -0.22686344385147095, -0.47865113615989685, 0.4320094585418701, -1.5183414220809937, -0.08126017451286316, -0.0057161240838468075, 0.21251675486564636, -1.8031977415084839, 0.6582239866256714, -1.2214360237121582, 1.4089871644973755, -0.048932913690805435, -1.6157042980194092, -0.2460009902715683, -0.7074852585792542, 0.7368583679199219, 1.455844759941101, 1.0380610227584839, -1.2563899755477905, 0.21509014070034027, -1.2273598909378052, -0.1125255674123764, 0.6156176924705505, -1.9845433235168457, 0.5685552954673767, -1.1875723600387573, 1.5254297256469727, 2.382331132888794, 1.9033464193344116, -0.7887043356895447, -0.6367846727371216, -0.2734595239162445, -0.9635707139968872, -1.6040980815887451, -0.2303629219532013, -0.2969920039176941, 0.5713731646537781, 0.5595067739486694, -0.20730073750019073, -0.6824341416358948, 0.21476218104362488, -1.2031925916671753, 0.6125459671020508, -1.5316866636276245, 0.22290684282779694, -0.4367538094520569, -0.752007007598877, -0.21138307452201843, 0.1411081850528717, 0.828644335269928, 1.1317692995071411, -1.3830405473709106, -0.9074162840843201, 1.5582722425460815, 0.8774210810661316, 0.39063647389411926, -0.15724827349185944, -1.3372670412063599, 1.260864019393921, 1.432470440864563, -0.4807504415512085, -0.9381535649299622, 0.4344381093978882, -1.1014842987060547, -0.33145126700401306, 0.04924003779888153, 0.05904185026884079, 1.6929908990859985, -1.3215773105621338, 1.1173839569091797, 1.0877559185028076, -0.9973536729812622, -1.1020129919052124, 0.09532126784324646, -0.29441750049591064, -0.41337552666664124, -0.893032431602478, -1.9340758323669434, 0.22322168946266174, 1.526160717010498, -1.476102590560913, -0.37803032994270325, -1.004107117652893, 0.31227344274520874, 2.034775733947754, -0.1909460574388504, -0.2471943348646164, -0.015204107388854027, -1.3413997888565063, 0.7490084767341614, -2.4365224838256836, 1.1352652311325073, -0.11428038775920868, -0.4920634925365448, -1.7957029342651367, -1.3525254726409912, 0.5999270677566528, -0.8671466112136841, 1.0425478219985962, -0.3232939541339874, -0.4314671456813812, -2.163900375366211, 1.766030192375183, -0.7092545032501221, 0.8596162796020508, -0.82317054271698, -1.6147569417953491, 1.2143586874008179, -0.17058472335338593, 3.3531386852264404, 0.8954993486404419, 0.0007619601092301309, 0.31975477933883667, 1.3615436553955078, 0.02999843657016754, 1.9674055576324463, 0.031099967658519745, 0.05548151955008507, 0.48858746886253357, -1.522793173789978, 0.09257598221302032, -0.43476107716560364, -0.5109245777130127, 0.006520203780382872, 1.168776035308838, -0.3044481575489044, -0.4113316833972931, 0.5911784172058105, -1.071894645690918, 0.24341614544391632, -0.591530978679657, 0.010935870930552483, -0.2149670571088791, -0.5771381855010986, -1.8493016958236694, 0.5577592849731445, 0.9294430613517761, -2.4626736640930176, -1.9056239128112793, -0.4416152834892273, 1.930387258529663, 0.32338789105415344, 0.1476789265871048, -0.29210540652275085, 1.5242336988449097, -0.46574392914772034, -0.14589886367321014, -0.3171641230583191, -1.3452627658843994, -1.1017142534255981, -0.33315616846084595, -0.0009221048094332218, -0.08046133071184158, 0.7493621110916138, -0.38257870078086853, -0.6399469971656799, 1.0584739446640015, -0.7758482694625854, 0.6472173929214478, -0.3309381902217865, -1.839665174484253, -0.5503853559494019, -0.7017503380775452, -0.8729912638664246, 0.15117725729942322, 0.18874609470367432, -1.1387325525283813, -0.2109207808971405, 0.7139601111412048, -0.2573179304599762, 1.9603270292282104, 0.18204937875270844, 0.9593632817268372, -0.36557766795158386, 0.7874550819396973, -0.6190552711486816, 1.0085610151290894, 0.16131526231765747, -0.5956258177757263, -1.2099010944366455, 0.3031919598579407, 2.2687056064605713, 0.18778535723686218, 0.2882310748100281, -1.7095022201538086, 1.2014837265014648, 1.1793240308761597, 0.44857484102249146, -2.422266721725464, -1.5700523853302002, 0.4297594726085663, -1.3501152992248535, -1.4450809955596924, -0.7315094470977783, -0.9859859943389893, 1.8687664270401, -0.26106759905815125, 0.4210759997367859, -1.144247055053711, -0.7537699937820435, -0.6504496932029724, -1.30643630027771, -1.361553430557251, 0.36261051893234253, -1.1883023977279663, -0.20263396203517914, 0.48397740721702576, -0.40977486968040466, 0.885354220867157, -0.4395338296890259, -1.327336311340332, 0.35165834426879883, 0.7249476313591003, 1.8095085620880127, -0.8547351360321045, 2.309572458267212, 0.9243065714836121, 1.0285725593566895, 0.22943830490112305, 1.1603996753692627, -0.989368736743927, 0.12622058391571045, 2.532010555267334, 0.7551933526992798, 0.11205396056175232, 0.6134999990463257, -1.585686206817627, 0.05637392774224281, 0.4941326677799225, -0.6918350458145142, -1.6433359384536743, -0.6263025403022766, 0.5684733390808105, -1.1068141460418701, 1.0862843990325928, 0.5607285499572754, -1.5347108840942383, -0.22588448226451874, 1.1382253170013428, 1.5953062772750854, -1.8811893463134766, -0.053241629153490067, 0.22025175392627716, -0.7121816873550415, 0.3971962332725525, 1.6181162595748901, 0.17951180040836334, 0.5797513127326965, -0.08845586329698563, 0.8915709257125854, 0.5777825117111206, 2.095716953277588, -0.7285788655281067, 0.23073533177375793, -2.300969362258911, 1.0279083251953125, 1.046800971031189, -0.3785754442214966, 1.0846450328826904, 0.7527322173118591, -1.4943360090255737, 1.0468369722366333, -0.01654641516506672, 1.023036003112793, -0.061628568917512894, 0.8143460750579834], [-0.8836953043937683, -0.12626175582408905, 0.6196545958518982, -1.317644476890564, 2.0148351192474365, 1.2071754932403564, -0.6914364099502563, 0.8661720156669617, -0.6636070013046265, 0.011072761379182339, -0.9405165910720825, -0.31565338373184204, 0.8307303786277771, 0.4410339295864105, -0.22345779836177826, -1.538689136505127, 0.7347729206085205, -1.1233665943145752, 1.4574995040893555, -1.6899783611297607, -0.6495575308799744, -0.5263136029243469, -1.1832516193389893, 0.567382276058197, -1.3199762105941772, 0.4871930480003357, 0.17561876773834229, 0.9283429980278015, 0.00960139837116003, 2.0221304893493652, 0.24279876053333282, -0.16730231046676636, -0.311835914850235, 1.1325128078460693, -0.22603876888751984, -1.1387017965316772, 1.2519818544387817, 0.04771079868078232, 0.11609448492527008, 0.43702101707458496, -0.34827950596809387, -0.1396782249212265, 0.4460000991821289, -1.0632966756820679, -0.41291776299476624, -0.4503399133682251, 0.023083463311195374, -0.35044991970062256, 0.2534661293029785, 0.32791247963905334, 1.5256321430206299, 0.24383383989334106, -1.6999306678771973, -0.5237017869949341, -0.4012232720851898, -1.1007462739944458, -0.7094013690948486, -0.04000081121921539, 2.0606648921966553, -2.32594895362854, -1.2574183940887451, -0.9533668160438538, 0.6894271373748779, 0.42875757813453674, 2.155348300933838, -0.9353736042976379, 0.7686855792999268, 0.38335931301116943, -0.19634635746479034, 0.26948922872543335, -0.1677856147289276, -0.19918806850910187, -1.4340218305587769, -1.184988260269165, -0.4312400817871094, 0.07393438369035721, 1.3449043035507202, 0.46866828203201294, 1.3577101230621338, 0.18226666748523712, -1.336702823638916, 0.0302545465528965, -1.063674807548523, -0.10376645624637604, -0.7455017566680908, 0.11948709189891815, 0.0015615380834788084, -0.42191532254219055, -0.7301589250564575, 0.7814456224441528, 0.15702146291732788, -1.2061996459960938, 0.5286745429039001, -0.3097052276134491, 0.2539532780647278, -1.6512243747711182, 1.82895028591156, 0.5083967447280884, -0.4463891088962555, -0.8888219594955444, 0.474899560213089, -1.1930935382843018, 0.49287840723991394, -0.21900296211242676, 0.06888863444328308, -0.4259681701660156, -1.2362829446792603, 1.5914195775985718, 0.6700506210327148, 0.722062885761261, 0.406728595495224, -1.3745464086532593, 1.2724488973617554, -0.8443562984466553, 0.9120364189147949, 0.32802265882492065, -1.3867034912109375, -0.947311282157898, 1.6941357851028442, 0.6496354937553406, 1.262302279472351, -1.232532262802124, 0.5197590589523315, -0.7706721425056458, 1.3459912538528442, 0.31342703104019165, -1.700045108795166, -1.1908591985702515, 0.5096976161003113, -0.1441536396741867, 0.868044912815094, 0.5900309085845947, 0.30980128049850464, 0.4875049293041229, 0.5553705096244812, 1.4339675903320312, 1.016174077987671, -2.0378153324127197, 0.3233692944049835, -0.4635799527168274, 0.13679565489292145, -0.3650689125061035, 1.3341909646987915, 0.7922961115837097, 1.407866358757019, -0.16133737564086914, -1.5099765062332153, -1.3766956329345703, 1.6137880086898804, -1.4363535642623901, 0.07349893450737, -0.08690255880355835, -1.1455055475234985, 0.27756381034851074, 0.18581148982048035, 0.5793554186820984, -1.1606354713439941, 0.1355305314064026, -0.8945244550704956, -0.6090392470359802, -0.47764062881469727, 0.7422788143157959, -0.17123636603355408, -0.7254661917686462, -1.4525713920593262, -0.4461236298084259, 0.7686114311218262, -0.32668209075927734, 1.3364636898040771, -0.27394717931747437, 1.6291207075119019, -0.9565063118934631, -0.7807899117469788, -0.5290272235870361, -0.1910489946603775, -0.48940011858940125, 1.4410828351974487, -1.2205172777175903, 0.1389644891023636, -1.2585535049438477, -0.6189808249473572, -1.7975335121154785, -1.1471962928771973, 2.7522642612457275, -0.5080661773681641, -0.719571590423584, 0.2555966377258301, 1.0207775831222534, 0.162501260638237, -0.33820104598999023, -1.326545000076294, -0.06951692700386047, -0.38346830010414124, 0.3921016454696655, 1.1527745723724365, 0.25859972834587097, 0.7093296051025391, -0.13923002779483795, -0.2976926565170288, 2.0273361206054688, -0.2810687720775604, 0.7042567729949951, 0.590411365032196, -1.5987896919250488, 0.6298527121543884, -0.4651659429073334, 0.337399423122406, -0.42491212487220764, -0.9614322185516357, 1.219927430152893, 0.6066650748252869, 0.24985817074775696, 1.350417971611023, 0.31482934951782227, 0.18684835731983185, -1.0546793937683105, 0.7303094267845154, -0.8820402026176453, -1.108373999595642, 0.34806346893310547, 0.8015264868736267, -1.4736329317092896, 0.9439727067947388, -0.5163148641586304, -0.8308424353599548, 0.47836941480636597, -0.02377162128686905, 0.20445013046264648, 0.12951722741127014, -0.8203513622283936, 0.2988884449005127, 1.2551825046539307, -0.8325682878494263, -0.6557872891426086, 0.39901605248451233, 1.524485468864441, 0.15647006034851074, -1.3930318355560303, 1.2662522792816162, 0.05347612500190735, -1.223249912261963, 0.26060032844543457, 2.0322389602661133, -0.13119271397590637, 0.27057212591171265, -0.04169309139251709, 0.316613107919693, 0.3577384054660797, -0.11444009095430374, 0.09888532757759094, -0.009849267080426216, 0.517479419708252, 0.563814640045166, 0.6436415910720825, -0.8337996602058411, -0.46018552780151367, 0.5875693559646606, -0.7099701762199402, 0.7100518941879272, 0.47873732447624207, -1.0216326713562012, 1.5918898582458496, 1.4045491218566895, -1.3463976383209229, -1.2161686420440674, 0.036070067435503006, 1.1588380336761475, 0.7778631448745728, -1.2984153032302856, -0.6225640773773193, -0.8093651533126831, -0.7619825005531311, -1.9837138652801514, 1.8454344272613525, 1.8747923374176025, 0.33562523126602173, -0.3781406879425049, -0.1864032745361328, 1.262261152267456, 0.6892878413200378, 1.50649893283844, 0.09236022084951401, -0.4735472500324249, -0.1408267766237259, 1.4726476669311523, 0.1806240826845169, 1.0509750843048096, 0.6522712707519531, -0.42446407675743103, -0.2925606966018677, -0.8228764533996582, 2.118410110473633, -1.3246403932571411, -0.3305976390838623, 1.3794941902160645, 1.954062581062317, 0.5070140957832336, -1.5990931987762451, 1.3365145921707153, 0.6604741811752319, 3.5024242401123047, -0.6081374287605286, -0.8870882391929626, -0.9497885704040527, -0.04359735548496246, 0.41520047187805176, -0.08100436627864838, -0.3650067150592804, -0.8779522776603699, 0.7676517963409424, 0.3744323253631592, -0.13089635968208313, -1.8018637895584106, 0.06301480531692505, -0.32646527886390686, 0.4681805968284607, -0.5717433094978333, 0.9604518413543701, 1.7636489868164062, -0.23732639849185944, -0.18724766373634338, -1.7674716711044312, 0.05574502423405647, 0.6932138204574585, -0.8975796103477478, 2.394580364227295, -0.3535109758377075, 0.20915132761001587, -1.5340181589126587, -1.467332363128662, 0.6016090512275696, 0.01829744502902031, -0.23971065878868103, -0.71776282787323, -0.689074695110321, 0.19375881552696228, 0.4791216254234314, -0.3738252520561218, -1.2810777425765991, 1.4669915437698364, -0.09928736835718155, 1.075835108757019, 0.9171529412269592, -1.545522689819336, -0.9000557661056519, 0.5673004984855652, -0.14162613451480865, -0.5262667536735535, 0.3335745930671692, 0.3593713641166687, -2.1965949535369873, 0.1501857191324234, 1.8294273614883423, -0.0428292378783226, 0.9528617262840271, 1.2315865755081177, 0.9305512309074402, -0.1370926797389984, -0.1987960785627365, -0.19481036067008972, -1.3089195489883423, 1.8414949178695679, -0.6405401229858398, -0.6508299708366394, 0.426533967256546, 0.10474235564470291, -1.313547134399414, -0.577290415763855, -0.7494249939918518, 0.39800336956977844, 0.4505476951599121, 1.9175310134887695, 0.9818801283836365, 0.1884101778268814, -0.9543702602386475, -1.1338297128677368, 0.7535797953605652, -0.25349724292755127, -0.4440612196922302, 0.8727179765701294, 1.5876890420913696, -1.6597974300384521, -0.6830152869224548, 0.33133360743522644, -1.0483499765396118, -0.7772534489631653, -0.03279615193605423, 0.6397263407707214, 1.2884787321090698, -2.860450029373169, 1.1710941791534424, -0.8248758316040039, 0.9083877801895142, 0.38081175088882446, 0.8980482816696167, -0.06928938627243042, 1.4322118759155273, 0.3302742540836334, -0.6558980941772461, 0.631356954574585, -0.4673953056335449, -0.40678057074546814, -0.29622921347618103, 0.42093679308891296, -0.615532636642456, -0.30668386816978455, -0.25959479808807373, 0.5580736994743347, 0.948425829410553, 1.061976671218872, -0.5638312697410583, 0.3202393054962158, 0.8175445795059204, -0.6221558451652527, 1.1779850721359253, 1.0478811264038086, -0.5530981421470642, 0.8152152895927429, -0.33476606011390686, 2.1873626708984375, -0.5886641144752502, 1.2702183723449707, 2.3233773708343506, 0.29877564311027527, 0.3125205934047699, 0.275481253862381, 0.3811227083206177, 1.9147173166275024, 0.15397076308727264, -0.8345257639884949, 0.014837049879133701, 0.6028117537498474, -1.5019136667251587, -1.0539536476135254, -0.9420920014381409, -0.3281833827495575, 0.05206052586436272, -0.5742306113243103, -0.12170454859733582, 1.3890388011932373, 1.2934757471084595, 1.8626408576965332, 1.114414095878601, -0.5521644949913025, 0.49769535660743713, -2.3456621170043945, -0.3054840564727783, -0.45899057388305664, -1.7366836071014404, 1.0717408657073975, 0.7827457785606384, -1.316391110420227, 0.5158695578575134, 0.5535021424293518, 0.3885771334171295, 0.1407884955406189, -0.8046122789382935, -1.3275845050811768, 0.5814349055290222, 0.453959584236145, 0.9191249012947083, -1.062507152557373, -0.332780659198761, -1.59756600856781, 0.8436193466186523, -0.997962236404419, -0.31261399388313293, -0.9868370890617371, 0.7190290689468384, 1.1891473531723022, 0.6595538258552551, 0.5953256487846375, -0.48285725712776184, -0.6550331115722656, 0.1553470641374588, -0.2249608337879181, 0.48157620429992676, -0.1626298874616623, -0.01808493211865425, -0.0672295093536377, -1.0705865621566772, 0.9513555765151978, 0.869045615196228, 0.70192950963974, 0.4324667751789093, -0.42107903957366943, -0.4407397508621216, -1.2019678354263306, 0.8494858741760254, 0.18233169615268707, -1.3018146753311157, -0.8838509321212769, 1.0055835247039795, 0.620995283126831, 0.9517261385917664, 0.06008046865463257, 1.2715024948120117, -0.8905842304229736, 0.5803792476654053, 1.9749411344528198, 1.2097418308258057, 0.11354319006204605, 2.180168628692627, -0.48538053035736084, -1.5183336734771729, -0.7012749910354614, 0.24214550852775574, 0.30618810653686523, -0.9967050552368164, 0.14475592970848083, -0.14339841902256012, -0.02365740016102791, -0.4365933835506439, 0.22544442117214203, 0.13796062767505646, 0.9585340023040771, 0.4234388470649719, 1.667677640914917, 0.5013134479522705, 0.3816930055618286, -1.732138991355896, 1.4624123573303223, 1.4561679363250732, -0.20019252598285675, -0.3418976962566376, 0.760918140411377, -0.7023655772209167, 1.2066096067428589, 1.3386908769607544, -0.31020697951316833, -0.8230606317520142, -0.23694302141666412, 0.6233713626861572, 0.41114795207977295, 0.9321771264076233, 1.2060556411743164, -0.16462735831737518, 0.06992117315530777, -1.274967074394226, -0.32460275292396545, 1.5402255058288574, 1.1201250553131104, -0.1462818831205368, -0.9541557431221008, 0.9910022020339966, -0.381777286529541, -0.2083226591348648, -0.3654402792453766, -0.136370450258255, -0.09571307897567749, -1.5859895944595337, -0.6637159585952759, -0.26076051592826843, -0.3796808123588562, -0.5800310969352722, 0.6119077205657959, -0.26574796438217163, 0.9146718978881836, -0.9506006240844727, 1.952301025390625, -0.036196768283843994, -0.2963317334651947, 1.9354175329208374, 0.1355305016040802, -0.9693331122398376, -1.7701293230056763, -0.22585560381412506, -0.259969562292099, 1.3604434728622437, 0.3560228645801544, -0.26883384585380554, 0.9095593094825745, 0.5727076530456543, -0.8640422224998474, -0.5808946490287781, -1.5272808074951172, -0.4080749750137329, -1.381766676902771, -0.7189685106277466, 0.8981721997261047, -0.8822591304779053, 2.1247143745422363, 0.06558535248041153, 1.9113423824310303, 0.17325036227703094, 0.19919951260089874, 0.5332770943641663, 2.3446764945983887, 1.1896864175796509, 0.0748276561498642, -0.893340528011322, -0.21236927807331085, 2.4138691425323486, 0.7185295820236206, 1.8186559677124023, -1.7913895845413208, -0.45004579424858093, 0.769616425037384, 1.242736577987671, 0.17665603756904602, 0.13725610077381134, 0.7721518278121948, -1.9282879829406738, 0.9671167731285095, 0.8002954721450806, -0.25452545285224915, -1.646856427192688, -0.28194180130958557, -0.8249812722206116, 0.1049664318561554, 0.5676740407943726, 0.46966275572776794, -1.8791308403015137, -0.7937951683998108, -0.07850421965122223, 2.6812238693237305, 0.3230225741863251, -0.586638867855072, -0.4400341510772705, -1.62313711643219, -1.0574100017547607, 0.48792263865470886, 0.30933406949043274, 0.8852564096450806, -0.9328939318656921, 0.44461938738822937, -0.23319122195243835, -0.4525068700313568, 1.006119728088379, -1.844401478767395, 2.2907748222351074, 0.6506516933441162, -1.7221232652664185, 0.7586534023284912, 0.49528247117996216, 1.191620945930481, -0.3977106213569641, -0.7796766757965088, -0.5955008268356323, -0.1256050020456314, 0.8907248377799988, -0.10340187698602676, 1.2620066404342651, 0.7267842888832092, -0.282009094953537, 0.23269608616828918, -0.9560911059379578, 0.6859878897666931, -0.20850884914398193, -1.783320665359497, 2.2552988529205322, -0.581061065196991, -1.5873770713806152, -1.3275915384292603, 1.1361863613128662, 0.895135223865509, 0.9623038172721863, 0.4962383806705475, -1.4733281135559082, -1.4302036762237549, -0.11241879314184189, -0.20067818462848663, -1.8098814487457275, -0.6995446085929871, -1.053420066833496, 1.0372170209884644, 0.2631567120552063, -0.2618432641029358, -0.25369739532470703, 0.03733394667506218, -1.6866596937179565, -0.38232263922691345, -2.50653338432312, 1.0562089681625366, 0.6318593621253967, 0.03245066478848457, -0.5114622712135315, 1.2788712978363037, 0.6214922070503235, -0.15029878914356232, -1.269797682762146, -0.0738808810710907, -1.7990964651107788, -0.2145080268383026, -0.6910684108734131, -0.6787385940551758, 0.4722650945186615, -0.3676016926765442, -0.5137544870376587, -0.5921258926391602, -1.0130707025527954, 0.6137964129447937, -0.8602162599563599, -0.45692941546440125, -0.34252694249153137, 0.472385972738266, -1.1997361183166504, -0.5084493160247803, -0.07823730260133743, 0.4591309428215027, -0.05136366933584213, 0.8782227039337158, 0.12763968110084534, 2.008676528930664, 0.8195434212684631, -0.4187052845954895, -0.4058573842048645, -1.879820466041565, -0.015634113922715187, 0.07195921987295151, -0.3320317566394806, -1.5599877834320068, -0.2766880691051483, 0.5941475033760071, -0.6712872385978699, 0.6790531277656555, -0.032641515135765076, 0.39195793867111206, -0.09312528371810913, -0.27194127440452576, -0.11501263082027435, -0.9073165655136108, -2.6270053386688232, 0.7600584030151367, 0.6860728859901428, -0.4907499849796295, -1.516768217086792, 1.03285813331604, -0.14817796647548676, -0.5971338152885437, 1.0203858613967896, 0.04027185216546059, -1.132678747177124, 0.2688237726688385, 0.12214469164609909, 1.2385845184326172, 1.1875170469284058, 0.04511439427733421, -0.21034371852874756, -1.1714023351669312, -0.5459038615226746, 0.3260895311832428, 0.36744388937950134, 0.8663221001625061, -1.8567756414413452, 0.5431037545204163, 0.9413545727729797, -0.5116531848907471, 0.5097512602806091, -1.5503456592559814, -0.32609033584594727, -0.5465570092201233, -0.5336235761642456, -0.09931633621454239, 1.1852353811264038, 0.007913450710475445, 0.70157390832901, -0.22746695578098297, 0.15721292793750763, 1.2680238485336304, -0.0841442197561264, -0.20534862577915192, 0.28727295994758606, -0.06699976325035095, 0.47114887833595276, 0.3866846263408661, -1.3746610879898071, -0.7586658000946045, -0.29176560044288635, 0.8765943050384521, -0.44283273816108704, 0.28300976753234863, -0.745643138885498, -0.8096293807029724, -0.45977240800857544, 1.8306857347488403, 0.2090909332036972, -0.28165769577026367, -2.587977170944214, -1.115497350692749, 2.6235861778259277, 0.532485842704773, -1.0516791343688965, -1.2681783437728882, -1.0720659494400024, 0.4251895844936371, -0.007417577784508467, -0.8079559803009033, 0.6068148016929626, 0.03411407023668289, 0.7708664536476135, 0.21519312262535095, -0.4481232166290283, -0.42492443323135376, -0.23108014464378357, 0.6800122261047363, 0.688932478427887, -0.9220885634422302, -0.9418125748634338, 0.4482969641685486, -2.715641736984253, -0.3646286427974701, 0.27268871665000916, 3.4786384105682373, -0.05581630766391754, -0.21522757411003113, -0.044846776872873306, 1.016990065574646, -3.1712136268615723, 0.3164283335208893, -1.7426785230636597, 1.6825264692306519, -1.9734420776367188, -0.07634592801332474, -1.0798834562301636, 2.341773271560669, 0.5734763145446777, 1.3066351413726807, 1.3070377111434937, 0.45751267671585083, -0.5406126379966736, -0.8601461052894592, 0.6794224381446838, -0.06447818130254745, -0.47701168060302734, -0.06832366436719894, -0.08970684558153152, 0.16845540702342987, -0.434821218252182, 0.5499938130378723, -0.9001472592353821, -0.1107105165719986, 0.35910362005233765, -0.395076185464859, -0.39914339780807495, -1.2372455596923828, -0.08640193939208984, 0.9220385551452637, -0.43316537141799927, -0.1367414891719818, 2.4798789024353027, -1.357791543006897, -1.0655546188354492, -1.0270055532455444, -0.40734660625457764, 0.5811289548873901, 0.0679340735077858, 1.7183253765106201, -0.5822509527206421, -0.5148244500160217, -1.4186928272247314, -0.03122766874730587, 0.050290368497371674, -0.5907392501831055, 0.43132150173187256, -1.8741837739944458, 0.6057078242301941, 1.3729774951934814, -0.051498621702194214, -0.2130453884601593, 0.09148313850164413, 0.8402425646781921, 0.684077262878418, -1.228832721710205, -0.5958269238471985, 1.4499324560165405, -0.5666952133178711, 0.6420334577560425, 0.2929675579071045, -0.2866418659687042, 0.5432265996932983, -0.9472718238830566, -0.8307638764381409, 0.13159622251987457, -0.5018054246902466, -2.2847533226013184, -0.7913803458213806, 0.03065858595073223, -2.0319082736968994, -0.45803016424179077, -2.367488384246826, -1.2145568132400513, 0.1447466015815735, 0.7929388880729675, 2.054673433303833, -1.3339868783950806, 0.5593554973602295, -0.8356783986091614, -2.5221269130706787, 0.17843930423259735, 1.2548855543136597, 0.707916259765625, 0.6509279012680054, 1.0535995960235596, -1.2023342847824097, 1.4917389154434204, 0.9429915547370911, 0.48609626293182373, 0.6819623708724976, 0.521976113319397, -0.6455932259559631, -0.21801577508449554, -0.20067918300628662, 2.196666955947876, 2.2393009662628174, 1.3771976232528687, -1.1657400131225586, 0.4578436017036438, -1.054973840713501, -0.09271369129419327, -0.7462060451507568, 1.3521610498428345, -1.1442188024520874, -0.1826716810464859, -1.4477125406265259, 1.6002390384674072, -0.4569939076900482, -0.4728010594844818, -1.1687179803848267, 1.7068082094192505, -0.04826787859201431, -0.7073372602462769, 0.7393249273300171, -1.0858234167099, -0.05243930220603943, 0.2958940267562866, 0.8971734642982483, 0.8179014921188354, -0.5198429822921753, -0.33790323138237, -0.8740450739860535, 0.6141513586044312, -0.647153377532959, 0.9257776737213135, -0.8731593489646912, -0.2740163803100586, -1.3303614854812622, -0.016214173287153244, 0.7882731556892395, -0.6217318177223206, 0.7134497761726379, -0.22839373350143433, 0.935054361820221, -0.6393467783927917, -1.89811372756958, 1.9295507669448853, -0.6638102531433105, -0.15551939606666565, 0.13927605748176575, -0.5773974657058716, 2.1530098915100098, -0.033203911036252975, -1.2226439714431763, -1.5916370153427124, 2.083984136581421, -1.1852504014968872, -0.37220504879951477, 1.4683126211166382, -0.9548286199569702, -0.6509631872177124, -0.04930053651332855, 2.1528983116149902, -0.22894684970378876, -1.6329845190048218, -1.8944685459136963, -1.287649393081665, 0.24886447191238403, 0.030578317120671272, -0.04858776554465294, -0.39183470606803894, 0.7545256018638611, -0.23946647346019745, -0.8888281583786011, -1.5569100379943848, -0.4107384979724884, 1.264635443687439, 1.0010337829589844, 0.16749052703380585, 0.47855913639068604, -0.9365962743759155, 1.9052178859710693, -0.07105983793735504, 0.4990719258785248, -0.7683502435684204, -1.27190363407135, -0.9190018177032471, -1.1803234815597534, -0.28237515687942505, -1.9613794088363647, 0.5088592171669006, 0.8306529521942139, -1.2188615798950195, 1.4906587600708008, 0.77720046043396, -0.12693670392036438, -0.4411492645740509, -0.5387264490127563, -0.1617506891489029, -0.6746430993080139, -1.285581350326538], [1.8737452030181885, 0.7291198968887329, -0.5996711850166321, 0.35807004570961, -1.106027603149414, -1.5294021368026733, -0.8048916459083557, 0.67112135887146, -0.8148232102394104, 0.43555912375450134, 1.1488211154937744, 1.108464002609253, -0.22177647054195404, -0.26921623945236206, -0.14800314605236053, 0.22901572287082672, 0.5913932919502258, 2.2864794731140137, 1.4927937984466553, -0.8753877878189087, -1.0200625658035278, 1.3115108013153076, 0.49208515882492065, -0.8959602117538452, -0.03218415006995201, 0.6640786528587341, 0.24826964735984802, -0.48451152443885803, -0.9983406066894531, 0.2843851149082184, 0.9083905816078186, 1.3731732368469238, -0.37923330068588257, 0.2759019136428833, 1.0728493928909302, -0.057183898985385895, 0.14933057129383087, 0.09490583837032318, 0.8638880252838135, 0.7197898030281067, 0.7913418412208557, 0.04686151444911957, 1.5388195514678955, 0.24598079919815063, 0.5326374769210815, -0.7958548665046692, -1.3812837600708008, 0.37415963411331177, -0.5961039066314697, 0.5524083375930786, 0.9116005301475525, 0.2690460681915283, 0.8390846252441406, -0.5082887411117554, -0.15911783277988434, 0.9211868643760681, 0.19634073972702026, 0.6230788826942444, 2.0209832191467285, -0.6330246925354004, 0.9495587944984436, -1.3041467666625977, -0.8121591210365295, 0.2663102447986603, -1.1159915924072266, 0.31306859850883484, 0.40186211466789246, 0.23374006152153015, 0.5144674181938171, -0.3134516179561615, -0.24588562548160553, -0.2406710982322693, -0.03455330431461334, -1.310068964958191, 0.8768495917320251, -1.313398838043213, -0.03672001138329506, 0.770073652267456, 0.03778950124979019, 2.5460078716278076, -2.0174918174743652, 0.08411698043346405, -1.7034235000610352, 0.6550955176353455, 1.986093282699585, -1.2437171936035156, 0.0924704372882843, -0.3051270842552185, -1.4155590534210205, 0.843574583530426, -0.9510531425476074, -0.750986635684967, 1.9990196228027344, 1.0011564493179321, 0.011135352775454521, 1.3990719318389893, -1.2787806987762451, 1.0093662738800049, -0.8775193095207214, -0.9899981021881104, 0.8754404187202454, 1.898416519165039, 0.7634929418563843, 2.5933749675750732, -0.27345049381256104, -2.373927116394043, 0.7168583273887634, 0.11400525271892548, -0.8149219155311584, 0.5794454216957092, -0.42272040247917175, -0.8422999382019043, 0.564443051815033, 0.1348549723625183, -2.2515976428985596, -0.2232782393693924, -0.6932545900344849, -0.9177595376968384, -2.4622390270233154, -0.16159822046756744, -1.0279872417449951, -0.4993736445903778, 0.6148070693016052, 0.5265082120895386, -0.09269847720861435, -0.24265247583389282, -1.8828729391098022, 0.48069053888320923, -0.7788149118423462, -1.2629011869430542, 0.3468184471130371, 0.708251953125, -0.40881800651550293, 1.103688359260559, 0.9085899591445923, 0.8478093147277832, -0.5262372493743896, -0.6844332814216614, -0.47546452283859253, 0.6443585753440857, 1.3551429510116577, 0.5026915669441223, 0.8059049844741821, 0.7010853290557861, -0.5070449709892273, -1.1886152029037476, 1.319239854812622, 0.06524398177862167, 0.6613202095031738, -0.10147619247436523, -1.5540052652359009, -0.5022022128105164, 2.0546913146972656, -0.5907142162322998, 1.466843843460083, 1.4877796173095703, 0.7587429881095886, -1.0446853637695312, 0.2445678859949112, 1.0474908351898193, 0.429978609085083, -1.4006438255310059, -0.9753398895263672, 0.6722481846809387, 0.5008377432823181, -0.1515979915857315, -1.4864630699157715, -1.4676786661148071, 1.3383991718292236, -0.0585777647793293, 2.547593832015991, 0.5992050766944885, -0.11316794902086258, 0.09435071796178818, -0.6613644361495972, 0.17766572535037994, -0.8921661376953125, -1.5932451486587524, -0.3931586742401123, 0.6163027882575989, -0.3474051058292389, -2.302757501602173, 0.012251587584614754, -0.1182003989815712, -0.3993782103061676, 0.8550360798835754, -1.8032726049423218, 1.4831466674804688, -0.5922576189041138, 0.9597288370132446, 0.34918108582496643, -1.425955057144165, 0.9373824000358582, -0.755793571472168, -1.0508185625076294, 0.5349524021148682, -0.010113530792295933, 1.612321138381958, -0.8796916604042053, 0.26532015204429626, -1.024897813796997, -0.32763010263442993, -0.2245800793170929, 0.4902248680591583, -0.6714243292808533, 1.7570576667785645, -0.7069622874259949, 1.0355359315872192, 0.4619901180267334, 0.3038431704044342, -0.25879284739494324, 0.14405497908592224, -0.9540296792984009, 1.112735629081726, -0.5028330087661743, 0.017398536205291748, -0.8294094800949097, 0.07477574050426483, 0.10164434462785721, -0.27095454931259155, 0.6587122678756714, 1.0013368129730225, -0.3917461335659027, -0.3444467782974243, 0.38518333435058594, 0.1656322032213211, -0.8362805247306824, 0.2944425940513611, 0.1440601795911789, -0.8155904412269592, -0.7876186966896057, -0.5562103390693665, 0.7133477926254272, 1.3346399068832397, 0.9857640266418457, -0.1746591031551361, -0.6576116681098938, 0.6732990145683289, -2.055121660232544, 0.9532209038734436, -2.1474082469940186, 1.3148698806762695, -1.1047167778015137, -0.11036112904548645, 1.175176739692688, 0.3884626626968384, 0.21926334500312805, 0.5279417634010315, -0.1719885617494583, 0.5342448353767395, -1.8017597198486328, -0.6211220026016235, 0.08331888169050217, -0.03097454272210598, -0.34642377495765686, -1.137778639793396, -0.9547196626663208, 0.13023705780506134, 0.171584352850914, 1.6328660249710083, -2.4597465991973877, -0.044631022959947586, 0.13657058775424957, -0.8300412893295288, -1.3746192455291748, -1.0666660070419312, 0.15937480330467224, -0.941101610660553, 0.08205464482307434, 0.1084061786532402, -0.6444371342658997, 0.6024569272994995, 0.4279541075229645, 1.4036351442337036, 0.6963233351707458, 1.1693872213363647, 2.342944383621216, 0.4180167317390442, -0.6794946789741516, 2.0386834144592285, 0.6501700282096863, -0.4151575565338135, -1.0942271947860718, -1.5815777778625488, 1.2154711484909058, -0.025922603905200958, -1.370703101158142, 1.1981840133666992, 1.7863787412643433, -2.179755449295044, -0.9440652132034302, 0.9956281185150146, 1.5573687553405762, 0.5772321820259094, 0.7924034595489502, -0.7789638638496399, -0.7666628956794739, 0.1873444765806198, 0.6647763252258301, -0.811466634273529, 0.5941699743270874, -2.1183412075042725, -2.3864033222198486, -1.1550363302230835, -1.2025585174560547, -0.8130192160606384, 0.43173521757125854, -0.4409087002277374, -0.7330858707427979, -2.573603630065918, 1.9733567237854004, 0.6522195935249329, 0.9888368248939514, 1.8956507444381714, 0.8129275441169739, 2.640475034713745, 1.6738882064819336, -2.047750949859619, 0.8501268029212952, -0.6962776780128479, 1.0534507036209106, 0.26522985100746155, -0.7026019096374512, -1.3364624977111816, -0.5038255453109741, -0.24283847212791443, -1.2538989782333374, -0.2772216200828552, -0.7986447215080261, -0.7419463992118835, -1.0744813680648804, -1.4104222059249878, 0.0778106153011322, -0.9886645674705505, 0.09238681197166443, 0.4002966582775116, 1.2452125549316406, 0.818105936050415, 0.8452460765838623, -0.2796013653278351, -0.9060357213020325, -1.2817693948745728, 0.37903594970703125, -1.5964226722717285, -0.04860124737024307, 0.8888980746269226, -0.5082584619522095, -0.3463842570781708, -2.555605411529541, 0.37485262751579285, -0.8402347564697266, -1.1522859334945679, 0.6940789222717285, 0.25330406427383423, 0.4818978011608124, 2.2641375064849854, 0.2736075222492218, -0.5032522678375244, 0.35127896070480347, -1.2254993915557861, 0.2168143391609192, -2.065396308898926, 0.8874328136444092, -0.8582785129547119, 0.6255676746368408, 1.4983853101730347, 0.09751607477664948, -1.180371642112732, 0.5600048899650574, -0.2754778563976288, -1.0277047157287598, -0.775352418422699, 0.8564910888671875, 0.8148975372314453, 0.22909511625766754, 1.6546751260757446, 1.3385967016220093, 1.1814954280853271, 0.1928681582212448, 0.8200143575668335, -2.710082530975342, 1.5861685276031494, 1.043317437171936, -0.3336021900177002, -1.3062299489974976, -0.7954666614532471, 0.5816541314125061, -0.8042110800743103, 1.3942914009094238, 0.2546831965446472, -0.8181511759757996, -1.01967191696167, 1.3382763862609863, -2.2472946643829346, 0.3104729652404785, 0.3653312921524048, -0.7607031464576721, 0.37070760130882263, -0.12163592129945755, -0.31426072120666504, 0.24176865816116333, 0.5376903414726257, -0.3851039707660675, -0.9021151661872864, -1.1817387342453003, -1.9783633947372437, -0.9450174570083618, -1.3354729413986206, 0.777123212814331, 0.18452861905097961, -0.24561166763305664, 0.4600417912006378, -0.021558811888098717, 0.4085921347141266, -0.32238373160362244, 0.5417093634605408, 0.6948975324630737, -1.498341679573059, -0.30852264165878296, 1.79087495803833, 1.2642780542373657, 1.8235933780670166, 2.2719149589538574, 0.6677554249763489, -0.3875570595264435, -0.4874614179134369, 0.9472899436950684, -0.026064680889248848, 0.6111111640930176, -0.7660053968429565, 0.9426534175872803, -1.7644319534301758, 0.19947923719882965, -0.03254830464720726, -0.31383687257766724, -0.46773067116737366, 1.2006337642669678, 0.6035221219062805, -0.1341736614704132, -2.4612460136413574, 0.05466241016983986, 1.7390037775039673, 0.23147344589233398, -1.5662310123443604, 0.3121384382247925, -1.2055072784423828, -0.7611093521118164, -0.9806892275810242, 0.7741273045539856, 0.40294137597084045, 0.8258913159370422, 0.5084062218666077, 0.557992696762085, -0.40111956000328064, -0.23328982293605804, -0.126112699508667, -1.139953374862671, 0.1359645277261734, 0.40938055515289307, 0.4388086199760437, 0.1919328272342682, -0.7896623611450195, 0.5689675807952881, -0.2976764738559723, -0.8306729793548584, 0.9125322699546814, -1.6828769445419312, 2.0564754009246826, 0.7363130450248718, 0.3675095736980438, 0.8966436386108398, -1.0652958154678345, -0.0924830436706543, 1.6138904094696045, -0.22155413031578064, 1.4140459299087524, 0.736535906791687, 0.5587695240974426, 1.0497578382492065, 0.7713073492050171, -0.3413369059562683, 0.058058712631464005, 0.06485405564308167, 1.6555249691009521, -0.7077533602714539, 0.28600114583969116, 0.14675021171569824, 0.15493939816951752, -0.8256862759590149, 0.2621100842952728, 0.6749285459518433, 1.2890172004699707, -0.4413466453552246, -0.206624373793602, 0.2903580367565155, -0.06116178631782532, -0.1134023666381836, 0.5563117265701294, -0.14881862699985504, 1.100142240524292, 0.5442603230476379, 0.2247159481048584, -1.160904049873352, 1.5522340536117554, 0.1748843640089035, -1.4633100032806396, 0.09013382345438004, 1.188181757926941, 1.089444875717163, 0.610287606716156, -1.472760558128357, 1.4374970197677612, -0.767989993095398, -0.5683841109275818, -0.9528141617774963, 0.9567652344703674, -2.1100926399230957, 0.2851699888706207, 0.38255417346954346, 2.219198226928711, 0.29405710101127625, 0.5019105672836304, 0.18011140823364258, -1.5074094533920288, 0.20782437920570374, -0.15818709135055542, 1.1717851161956787, -1.1957595348358154, 3.5223419666290283, -1.545610785484314, 1.2376291751861572, 0.27690863609313965, -1.357938289642334, 0.250024676322937, 0.26878657937049866, 0.3527890741825104, 0.16341336071491241, 0.36965832114219666, -0.8457869291305542, -2.199845314025879, 1.4061174392700195, 0.8772826194763184, -0.8907009959220886, -0.6490747928619385, 1.0861965417861938, 1.5587742328643799, 1.854286789894104, 1.6917433738708496, -1.8873519897460938, 0.43438321352005005, -0.5678458213806152, 0.36628443002700806, -0.18030743300914764, -0.15299107134342194, -0.5172438025474548, 0.8996573090553284, 1.2889461517333984, 0.8513956665992737, 0.7653282880783081, -0.29510027170181274, 0.44894886016845703, 0.12497424334287643, 0.027665140107274055, -0.33916759490966797, -0.5342745780944824, 0.7508541941642761, 0.7931572198867798, -0.3689032196998596, 0.8451396226882935, -0.35937926173210144, 1.629809856414795, -2.127459764480591, -0.9885806441307068, 0.16145317256450653, -0.8481371402740479, -0.4693153202533722, 0.2304421216249466, 1.1438361406326294, 0.9130726456642151, 0.007919196039438248, 0.6753389835357666, 0.343667209148407, 1.306604027748108, 1.4667837619781494, -0.2634269595146179, -1.0592517852783203, 0.22251862287521362, 0.11202309280633926, -0.18622834980487823, -0.03535960987210274, 0.46601006388664246, 0.09832198172807693, 0.26953038573265076, 0.332395076751709, 0.8979688882827759, -1.9222726821899414, -0.3103831708431244, -0.34937164187431335, -0.9691314697265625, -1.5497642755508423, -0.09749724715948105, -1.3629175424575806, -0.9949027895927429, 0.16119562089443207, -0.7247602343559265, -0.549933135509491, 1.0005701780319214, 1.6612335443496704, -0.9621995091438293, -1.3126307725906372, -0.47873809933662415, -0.6093453764915466, -0.11419704556465149, 1.448008418083191, 1.346096396446228, 0.823605477809906, 0.3513143062591553, -0.5295137763023376, -0.2131194919347763, -0.717680811882019, -0.1256328821182251, -1.1556087732315063, -0.7552228569984436, -1.9129328727722168, 0.01840379647910595, 1.4831523895263672, 0.013753736391663551, -1.8517041206359863, 1.7881733179092407, 1.654181718826294, -1.1303563117980957, -1.013437271118164, 1.1347311735153198, 1.223576307296753, 0.3493969142436981, 0.5245543718338013, -0.6370130777359009, -0.027821334078907967, -0.8744156956672668, 0.8118334412574768, -0.06104103848338127, -0.5072388052940369, 0.5002886056900024, 1.701460838317871, -0.5595461130142212, 0.9411829710006714, -0.30373117327690125, -0.14887858927249908, 0.07306690514087677, -0.5651913285255432, 0.25506407022476196, 0.9058681130409241, 2.351231813430786, -1.0835866928100586, 1.1637545824050903, 1.377866506576538, 0.4496541917324066, 0.26936858892440796, -0.31468358635902405, -0.09956472367048264, 0.15877656638622284, 0.3547155261039734, 0.06083160266280174, 0.4562699496746063, -0.8993512988090515, 1.922747254371643, -1.189056396484375, 1.1696995496749878, 0.19182062149047852, -0.9199817776679993, -0.18638871610164642, 1.1636083126068115, -0.45498397946357727, -1.0202605724334717, 0.9038807153701782, -0.5193167328834534, -1.6906880140304565, -1.149445652961731, 1.2918660640716553, 0.08123341202735901, 0.7874011397361755, 1.2422305345535278, -0.2795560657978058, 0.7450344562530518, -0.24650157988071442, -0.19250835478305817, 1.01984441280365, 0.3618422746658325, -0.9405595064163208, 0.6456051468849182, -1.188852071762085, 0.6357314586639404, 0.8659769296646118, -0.6910445094108582, -0.7215179800987244, -1.142287254333496, 1.293771743774414, -0.16678377985954285, -1.113627552986145, -1.3360402584075928, 0.7692034840583801, -1.4912705421447754, -0.1844651997089386, 0.6483106017112732, 0.20321524143218994, 0.7452574968338013, -2.22983717918396, -0.027781175449490547, -0.3619749844074249, 1.5625388622283936, 1.3682180643081665, 0.004457780160009861, 0.23706774413585663, -0.2119399756193161, 1.7529276609420776, -0.7298775911331177, 0.16427774727344513, -0.04988281801342964, -3.7506208419799805, 0.2959105372428894, 0.36217641830444336, 0.27421408891677856, 0.3674519956111908, 2.078200578689575, -0.5946452021598816, -0.03447824716567993, -1.1128928661346436, -0.9967344403266907, 0.17510777711868286, 0.18668422102928162, -1.3601018190383911, 0.20700779557228088, 0.9748560190200806, -0.5979034304618835, -0.42606276273727417, -0.7206680178642273, 0.047833334654569626, 0.980266273021698, -1.2543766498565674, 1.0922257900238037, 0.21268415451049805, 0.33162713050842285, 0.9843103289604187, 1.0009516477584839, -0.44717657566070557, -0.45775339007377625, -0.0757935494184494, -1.907236099243164, 0.6597397923469543, 1.7818620204925537, -2.506436824798584, 0.9201370477676392, 0.02449958398938179, -0.12201552838087082, 1.5953638553619385, 0.5795025825500488, -1.7645541429519653, 1.135164499282837, 0.13249194622039795, 0.9199600219726562, -0.0046777441166341305, 0.8528397679328918, -1.2351102828979492, 0.14924539625644684, 0.946566104888916, 0.6236945986747742, 1.2107864618301392, 0.7005419135093689, 0.15448865294456482, -0.5255701541900635, -0.137550950050354, 1.8619225025177002, 0.8261342644691467, 0.4993895888328552, 0.7730386257171631, -0.9295066595077515, -0.45232149958610535, -0.058495666831731796, 0.802886962890625, -1.3242741823196411, 0.5721263885498047, 0.866409420967102, 0.5901235938072205, 0.22855226695537567, 0.3087851405143738, 1.1746524572372437, -0.6394059062004089, -0.46931520104408264, 0.1648324579000473, -0.97818523645401, -0.5997083783149719, 2.22609543800354, -0.9258338212966919, 0.004150460474193096, 0.7207010984420776, 0.7880347371101379, -0.16497553884983063, 1.123958706855774, -0.816775381565094, -1.242243766784668, 1.181959629058838, -0.1087624654173851, 1.0337095260620117, -1.1395022869110107, 1.3997247219085693, -0.007198079489171505, 0.27042892575263977, 1.3320786952972412, 0.8978898525238037, 1.053056001663208, 0.60016268491745, -0.640875518321991, -0.05400695651769638, -1.6542906761169434, -1.7566593885421753, -0.059678010642528534, -0.5268423557281494, -1.3558789491653442, 0.5604153871536255, 0.8514466881752014, 0.09302501380443573, -1.2290680408477783, -0.6812005043029785, 0.72031170129776, 1.9116172790527344, -2.3958725929260254, 1.1896039247512817, -0.8697405457496643, 0.6791145205497742, 1.26483154296875, 0.3513137400150299, 0.36027246713638306, 0.8791258931159973, -1.1684389114379883, 1.029889702796936, 1.1664830446243286, -1.7387409210205078, 0.43594038486480713, 0.6392940878868103, -0.968968391418457, -1.0543632507324219, 1.344650149345398, -1.6245099306106567, -0.06169482693076134, 0.3526034951210022, -0.9270155429840088, -1.2088634967803955, -0.4440981149673462, 2.541553258895874, 0.758808434009552, -0.586051881313324, -1.4106934070587158, 0.6347824931144714, 0.16821445524692535, 0.36865630745887756, -1.2280017137527466, -1.8322265148162842, 1.3581546545028687, -0.39008283615112305, -1.0898021459579468, 1.2694185972213745, -0.8901930451393127, -0.6715736985206604, 0.09530878067016602, 1.548403263092041, 0.45902904868125916, -0.5264846086502075, -0.7320088744163513, -0.06813362240791321, 1.8728362321853638, 0.3588956892490387, 0.3545432984828949, -1.1223407983779907, 0.2767830789089203, -0.5740317702293396, -0.17302486300468445, -1.1428172588348389, -0.910235583782196, 1.11378014087677, -0.2656257450580597, 0.031061440706253052, 0.26420366764068604, -0.4560297131538391, 0.6245933175086975, -1.588276982307434, 2.1252334117889404, 1.7896140813827515, -2.5861175060272217, 1.046094298362732, 2.0138492584228516, -0.8945798873901367, -1.3386666774749756, 1.011792778968811, 1.0332366228103638, 1.1896251440048218, -1.3177303075790405, -0.7016255259513855, 1.0779043436050415, -1.1871440410614014, 0.48751214146614075, 0.33847668766975403, -0.13647828996181488, -1.333324909210205, -1.5392512083053589, 1.2414112091064453, 1.4824252128601074, 0.41537314653396606, 0.8363437056541443, -0.6029302477836609, -1.1707751750946045, 0.1941300928592682, -0.7086465358734131, -0.0501682311296463, 0.10820319503545761, -1.0629167556762695, -0.6272140741348267, 1.5262871980667114, 0.07527739554643631, 0.6341344714164734, 1.0827782154083252, 0.4878823459148407, -1.4891949892044067, -0.8611103892326355, 1.0973771810531616, 0.3748535215854645, -1.4822018146514893, 0.5273429751396179, -0.8525689840316772, 1.1816221475601196, -1.039536952972412, -0.686564564704895, -0.5175228714942932, 0.0356634147465229, 1.5208871364593506, 0.5739001035690308, -0.2701379358768463, -1.9384765625, -0.8169911503791809, 0.521832287311554, 0.26804834604263306, 0.011399329639971256, -0.35320883989334106, 0.9738795161247253, 1.3083078861236572, -1.395087480545044, -1.0604625940322876, 1.6830836534500122, 0.45558568835258484, -0.38829004764556885, -0.7381018996238708, 0.3499186038970947, -1.085351824760437, 1.8737024068832397, 0.24045740067958832, 0.9813409447669983, 0.14911788702011108, -1.6195578575134277, -0.7165766358375549, -0.38796910643577576, -0.17233771085739136, -0.8287169337272644, 0.3007245361804962, 1.615212082862854, -1.600146770477295, -0.9424689412117004, -0.14861847460269928, 2.1645429134368896, 0.058810796588659286, -0.3826451003551483, 0.9636534452438354, -0.35853296518325806, -1.5758169889450073, -0.4753429591655731, -1.3104298114776611, 0.2279909998178482, 0.07528439164161682, -0.5125166177749634, -0.019014442339539528, 0.33344021439552307, -0.7922181487083435, 1.1428723335266113, -0.3288082182407379, -0.7065620422363281, 0.696377158164978, 0.2648658752441406, -0.27900567650794983, 0.7091031670570374, -1.0842127799987793, -0.7897838354110718, -0.37153685092926025, -0.00397298252210021, 0.9904499053955078, -1.6093477010726929, 0.7083475589752197, 1.7531615495681763, -1.1733983755111694, -0.49237826466560364, 0.9152893424034119, 0.8322965502738953, 0.5430553555488586], [-1.854224443435669, 2.362039804458618, 2.3297507762908936, 0.3328368663787842, 0.5359857082366943, 0.26327431201934814, -0.2798251211643219, 0.47644665837287903, 0.07533835619688034, -0.25037989020347595, 0.305385947227478, -0.2624722421169281, 1.8787857294082642, 0.7901659607887268, -0.24017134308815002, -0.3646937608718872, 0.6994169354438782, -0.4557250440120697, 1.2917160987854004, -0.29495474696159363, -1.380614995956421, -0.5518296957015991, 1.0465998649597168, -0.5327154994010925, -1.2311166524887085, 1.228575587272644, -1.3051815032958984, -0.09009762108325958, 1.2530747652053833, 0.5117831826210022, -0.40133407711982727, -0.03227223455905914, 0.9990277290344238, -0.6925159692764282, -0.2937237024307251, 1.0013386011123657, -0.6230538487434387, 0.865158200263977, -0.44972386956214905, 1.510111689567566, -1.5915497541427612, -0.5102606415748596, 0.06505784392356873, -0.2896623909473419, -0.5294049382209778, -0.248099684715271, 2.7121243476867676, -0.9414410591125488, 1.2432000637054443, 1.3443119525909424, 1.840871810913086, 0.5467589497566223, 0.8157642483711243, -1.496201515197754, -1.41680109500885, -0.5593195557594299, 0.6698703765869141, -0.6025020480155945, -0.5672282576560974, -0.527982771396637, -1.3579427003860474, 0.28647151589393616, 0.7845517992973328, -2.2446539402008057, 0.0019228683086112142, 0.2909620404243469, -0.39311540126800537, 0.7135083675384521, 0.5395460724830627, -0.6583045721054077, -0.8201888203620911, 1.0045474767684937, -0.01322871632874012, 0.3324161171913147, -0.5905666351318359, 1.4226583242416382, -0.36672738194465637, 0.4691030979156494, -1.0565763711929321, 0.721368670463562, 1.277414083480835, -1.6586406230926514, 0.02132236585021019, -1.0359466075897217, 1.2838846445083618, 0.4729018211364746, 0.4980847239494324, -1.8658115863800049, -0.299562007188797, -0.8560904264450073, -1.1872830390930176, 0.3460516035556793, 0.028700297698378563, 1.1458395719528198, 1.812920331954956, 0.14059890806674957, 0.7543483376502991, -0.4450245797634125, 0.46792498230934143, 2.500872850418091, 0.6212683916091919, 0.01617409475147724, 0.6726474165916443, 1.1818729639053345, 1.2169420719146729, 0.1304849088191986, -0.9496193528175354, -0.8472057580947876, -1.970749020576477, 0.8569473028182983, -0.5312891006469727, -1.54385244846344, 0.5329227447509766, -2.032548189163208, -1.1223971843719482, 1.136383056640625, -0.6078003644943237, -1.3121657371520996, 0.15078407526016235, 0.6535086035728455, -0.5895391702651978, 0.3424852192401886, -0.24955469369888306, 1.880785346031189, -0.05848516896367073, -1.9582358598709106, -0.38272783160209656, 2.748910903930664, 1.0918983221054077, -0.5765140652656555, -1.176061749458313, -0.7933520674705505, -0.44209396839141846, 0.051360633224248886, 0.7501412630081177, -0.57921302318573, -1.9279745817184448, -1.2490276098251343, -1.2993035316467285, 0.05311468988656998, -0.6989889144897461, -1.2539646625518799, 0.45385223627090454, -1.450745940208435, 1.853012204170227, 0.19021070003509521, -0.40112531185150146, -2.219550132751465, -0.20045672357082367, 0.9044784903526306, -0.7414699792861938, 1.2115908861160278, 0.9568835496902466, 1.2944047451019287, 0.5773850083351135, 1.0958867073059082, 1.2887604236602783, -1.6756491661071777, -0.3646351993083954, -0.2811621427536011, -0.7819153666496277, 0.44285473227500916, -0.36192700266838074, -1.3518619537353516, -1.4251055717468262, -0.3038332462310791, -0.7589373588562012, -0.24016998708248138, -0.3224993944168091, -0.16023775935173035, 0.8997126221656799, 0.5828240513801575, -0.5202568769454956, -0.7016947269439697, -0.8290373682975769, 0.12400399893522263, -0.7792888283729553, 0.6046850681304932, -1.0869219303131104, 0.05210969224572182, -0.31976303458213806, -1.4465088844299316, -1.0663424730300903, 0.9083476066589355, -0.5185222625732422, -1.4973788261413574, 1.711029052734375, 0.18089914321899414, 0.5072816610336304, 0.7342672944068909, -0.18414491415023804, -0.05292551591992378, -1.2975435256958008, 0.9240834712982178, -0.00355786201544106, -1.467539668083191, 1.9299445152282715, -0.4241706132888794, -0.366672158241272, 0.33810749650001526, 0.10836589336395264, 0.25408732891082764, 0.1524258255958557, -1.6202116012573242, 0.6344475746154785, -0.687103807926178, -0.4331761598587036, -0.5449445843696594, 0.24101395905017853, -0.7008891701698303, -0.0803985446691513, 1.0933382511138916, 1.3907814025878906, 1.7096221446990967, -0.8807893395423889, -0.10928677767515182, -1.7069430351257324, 0.688371479511261, 0.21679437160491943, -0.3505138158798218, 0.4569408595561981, 0.6910094618797302, -0.427318811416626, 0.008868083357810974, -0.5196818709373474, -0.3436630368232727, 0.2955361008644104, -0.5807260870933533, 0.034378618001937866, 2.9168851375579834, 2.5650904178619385, -0.26171383261680603, -1.9141285419464111, 0.16500437259674072, 0.20385698974132538, -1.1680400371551514, -0.12825562059879303, 0.06942863762378693, -0.906981885433197, -0.1454382836818695, 0.9476553201675415, 1.1397868394851685, -0.12389814108610153, -0.7062733173370361, 0.8911926746368408, 0.4153854250907898, 0.716284990310669, 1.6341320276260376, -0.5044506788253784, -1.130629301071167, 1.1088868379592896, 1.120465874671936, 1.4242022037506104, 1.6152000427246094, -0.7413103580474854, -0.3800312280654907, -0.2176564335823059, 0.5042084455490112, 1.3665778636932373, -1.0445301532745361, 0.36641725897789, 0.3159462511539459, -1.2816706895828247, 0.22636207938194275, 0.3072986304759979, 0.1358724981546402, -2.0789670944213867, -0.059697654098272324, -0.7967281937599182, 0.19279685616493225, 0.07072044909000397, 0.034333862364292145, 0.6579593420028687, -0.46868520975112915, -1.0376054048538208, -1.8157072067260742, -0.2445385754108429, -0.4797748327255249, -1.1703680753707886, -0.426611989736557, -0.3157043159008026, -0.3652309477329254, 1.3906551599502563, -0.03962983936071396, 0.025509128347039223, -1.171739935874939, -0.9022650122642517, -0.6643670201301575, -0.5053341388702393, 1.763360619544983, -0.7916986346244812, -1.8122919797897339, 1.8829855918884277, 0.8848848342895508, 0.7901281118392944, -0.24683231115341187, -0.010782393626868725, -0.7725029587745667, -1.153003215789795, 1.5559570789337158, -0.5268287658691406, 0.3000125586986542, 0.0383259579539299, -1.362424612045288, 0.4081539511680603, 2.388770341873169, 1.6740370988845825, -1.0860711336135864, 1.6442500352859497, 0.23218950629234314, -0.8960049748420715, 0.6030701994895935, 0.3801117539405823, 2.037003993988037, 1.4253973960876465, -0.6558573246002197, 0.3958693742752075, -0.9839866757392883, -0.9316397905349731, 0.1352163404226303, -1.545886516571045, -0.052927590906620026, 0.5928663611412048, 0.47831740975379944, -0.6395667195320129, 0.20449993014335632, 2.8403992652893066, -1.576071858406067, -0.6032772660255432, -2.2308318614959717, -0.8995164036750793, 0.23433436453342438, 0.22850994765758514, 0.20837262272834778, -0.6045209765434265, -0.029628144577145576, -0.7172871828079224, 0.32274696230888367, 0.1326724886894226, 0.6220396757125854, 1.4190551042556763, 0.053669724613428116, 0.5582161545753479, 0.4283113181591034, -0.0261390283703804, 0.45880356431007385, -0.38506823778152466, -0.36017313599586487, 0.013606863096356392, 2.2421481609344482, -0.8443921804428101, 1.5236713886260986, 2.125566244125366, -1.5207099914550781, 0.3567304015159607, -1.3471585512161255, 0.586540162563324, 0.4741598665714264, 1.5138434171676636, 0.17619919776916504, -1.132346510887146, -0.218723863363266, 0.039026692509651184, 0.8103602528572083, -0.794120728969574, -1.4059337377548218, 1.4264960289001465, -0.3020242154598236, -0.14126740396022797, 1.4940623044967651, -0.7384267449378967, -0.14464445412158966, -0.5141178369522095, 1.1796026229858398, -0.49754464626312256, -0.05408093333244324, -0.28847044706344604, 1.0377827882766724, 0.4315154254436493, 0.6808634400367737, -0.42709288001060486, -1.4016244411468506, -0.7522410154342651, -0.13659216463565826, -0.43903011083602905, 0.8369160294532776, 1.0189989805221558, 0.6026085615158081, -1.507462501525879, 1.4646756649017334, 0.458224356174469, 1.7254469394683838, 0.5970233082771301, 0.1178373321890831, 1.601632833480835, -0.23227234184741974, 0.9153119921684265, -0.2653447985649109, -0.5014353394508362, -1.1504210233688354, 0.7461785078048706, -0.4393698573112488, 1.4030160903930664, 0.19247126579284668, -1.6327581405639648, 0.9244004487991333, -0.7165163159370422, 0.3563188314437866, 1.5078250169754028, 0.5897659659385681, -1.681807518005371, -1.2795830965042114, 0.6365281343460083, 1.2936245203018188, 1.2494020462036133, -2.5126991271972656, 1.0724745988845825, -0.3969078063964844, -1.5321121215820312, -1.5655083656311035, -0.6137297749519348, 0.1976289600133896, 0.5714439153671265, 0.5661506652832031, 1.0548186302185059, -1.4054101705551147, 1.1891053915023804, -1.1135337352752686, 0.09993503987789154, 0.24711738526821136, 0.22561192512512207, 1.2878000736236572, 0.9123519062995911, -0.9986217021942139, -0.002067301422357559, 1.3219648599624634, 2.210125207901001, -0.28530818223953247, 0.7709079384803772, -1.8095961809158325, -0.6983821988105774, 1.1172056198120117, -0.44712644815444946, 0.6900849938392639, -1.0630520582199097, -0.30298060178756714, -2.355123519897461, -0.03518610820174217, 2.249786376953125, 0.06758091598749161, -0.4860789179801941, 0.68093341588974, 0.1433933526277542, 0.9846422672271729, -3.031304359436035, -0.43860459327697754, -0.321576863527298, -0.07782560586929321, -0.32974016666412354, 0.9293506741523743, 0.117493636906147, -0.11799894273281097, -0.957339346408844, 0.6460901498794556, -2.016706943511963, -0.13520902395248413, -0.03260399028658867, -0.4697757959365845, 1.5803377628326416, -1.2694849967956543, -1.5823743343353271, 1.2805060148239136, 0.8937628865242004, 0.3560413718223572, 0.9737850427627563, 0.4390398859977722, 0.8453173041343689, 0.019134951755404472, -0.04375327005982399, -0.0873207300901413, 1.307916283607483, -0.37419140338897705, -0.5922031998634338, -0.07906679064035416, -0.4477745592594147, 0.7303635478019714, -0.22279319167137146, -1.6506128311157227, 0.39061859250068665, -1.4481269121170044, -1.2119344472885132, 1.242801547050476, -0.0384872741997242, 0.10579860210418701, 0.0659896582365036, 0.08246006816625595, 0.6057083606719971, -0.4990249574184418, -0.5311809778213501, -1.124977946281433, -1.5200629234313965, 0.38804176449775696, 1.5471042394638062, 0.20021897554397583, 1.7924343347549438, -1.0412966012954712, 0.5196822881698608, -0.9699812531471252, -1.1588411331176758, 0.8682909607887268, -0.8082653880119324, -0.49618977308273315, 0.3462893068790436, 0.5641733407974243, 0.13734790682792664, 0.8081430792808533, -0.7400351166725159, 0.43179404735565186, 0.8558101058006287, -2.1061434745788574, 0.3720257580280304, 0.6942050457000732, 1.217272400856018, -0.44380465149879456, -1.945073127746582, 0.1467633992433548, 0.9153527021408081, -0.5414685606956482, 2.4685165882110596, 2.5482077598571777, 0.056433141231536865, 1.0908094644546509, 1.7721686363220215, -1.9400256872177124, -0.176893949508667, 0.4544902741909027, -0.1969357430934906, -1.846595048904419, 0.82673579454422, 1.1181081533432007, 1.2459195852279663, 0.1317680925130844, -0.38895300030708313, 1.1015478372573853, -1.3102830648422241, -0.867365300655365, 1.5319187641143799, 0.842811644077301, -0.2829672694206238, -0.6334571242332458, 0.5274494290351868, 2.214118719100952, -0.10522783547639847, -0.9127485752105713, -1.4699369668960571, -1.543558120727539, 0.1433086395263672, 0.1410994529724121, -0.34006139636039734, 0.17838560044765472, -0.023518605157732964, -0.14411836862564087, -0.10160962492227554, -1.1364307403564453, 0.997591495513916, -1.6200547218322754, 0.2067854255437851, 0.20707854628562927, 0.3694901764392853, -0.7002573609352112, -0.17183250188827515, 1.498839020729065, -0.3497072160243988, -0.47119811177253723, 1.3193788528442383, -0.9582213759422302, -0.0030946340411901474, 1.1087638139724731, -1.4743402004241943, -1.401694655418396, 1.2727668285369873, -0.2704099714756012, 1.3641973733901978, -0.8768600225448608, -0.5854210257530212, 1.0505212545394897, -0.7285435199737549, -0.3877694308757782, 1.13167405128479, 0.41141581535339355, 1.1157523393630981, 1.9293193817138672, -0.48863404989242554, 0.0012855792883783579, 1.5875920057296753, -0.1354319304227829, 2.283607006072998, 0.7791032791137695, -1.6148488521575928, 0.3633112609386444, 0.23456613719463348, 0.974425733089447, 0.45848849415779114, 1.0244286060333252, -0.10173455625772476, 1.1445395946502686, -0.3393637239933014, -1.027182698249817, -1.2647802829742432, 0.02155320532619953, -0.3756297528743744, 0.17168158292770386, -0.18216902017593384, -1.2669479846954346, 0.014777442440390587, 0.9106634259223938, -0.16790418326854706, 0.1281014382839203, 2.3079171180725098, -0.6076154112815857, -1.7721959352493286, -0.0033172171097248793, 0.7979966402053833, -1.2721548080444336, -0.13503769040107727, 1.876559853553772, -0.12055559456348419, 0.7738337516784668, 0.04652947559952736, 0.7021233439445496, -1.1813851594924927, -2.309872627258301, 0.22626899182796478, 2.111637592315674, 1.6835159063339233, 0.5042433142662048, 1.3220865726470947, 0.06857805699110031, -0.8369140028953552, -0.5228655338287354, -1.4211570024490356, -0.34628912806510925, -0.17820966243743896, 0.9862743616104126, 0.8298584818840027, -1.7867205142974854, 1.0515440702438354, 1.146396517753601, 0.24211359024047852, 0.19475647807121277, -1.8675254583358765, -1.0371309518814087, -0.228899285197258, 0.166327103972435, 1.541662573814392, 1.2359163761138916, 1.1359970569610596, 0.425878643989563, -0.5340747833251953, -0.7959315776824951, -0.8823422789573669, -0.5331218242645264, 1.4784208536148071, -0.051462966948747635, -0.02188475988805294, -1.8164751529693604, 0.4228239357471466, 0.42399466037750244, -0.3317440450191498, 0.42689457535743713, -0.07893316447734833, -0.4831063449382782, 1.0385922193527222, 0.4388943314552307, -0.010477310977876186, 0.6701818704605103, -0.2986793518066406, 0.9638031125068665, -0.011960663832724094, 0.26429757475852966, 2.0108642578125, 1.3065226078033447, 1.6770931482315063, 1.5044816732406616, 0.4568330645561218, -1.0427268743515015, 1.661896824836731, -2.1495068073272705, -0.6159226298332214, -0.5696969628334045, 0.7241474986076355, 1.1075457334518433, 0.14653338491916656, 0.6417097449302673, -0.13952945172786713, -1.1916210651397705, 0.15692126750946045, -0.9133349061012268, -0.3682903051376343, -1.2566323280334473, -0.23542840778827667, -0.5252189040184021, 0.8355576992034912, 0.711596667766571, 1.5306719541549683, -0.04894590005278587, -0.27540239691734314, 1.3650728464126587, 1.591212272644043, -1.1809415817260742, -0.9229867458343506, 0.8831787109375, -0.6565451622009277, 0.2701084017753601, 0.3230404853820801, 1.1625475883483887, -0.2981424629688263, 0.6184073686599731, -0.20113763213157654, -0.881788969039917, 0.5331671833992004, 0.4106343388557434, 0.35521337389945984, 1.462038516998291, 0.041733406484127045, -1.4869624376296997, 0.023501388728618622, -0.5699203014373779, -0.7591404318809509, 0.9760475158691406, -0.4920442998409271, -2.2193195819854736, -0.4005435109138489, -0.058453600853681564, 1.4851053953170776, -1.3520034551620483, -1.7772465944290161, 0.6142017841339111, -0.37822675704956055, 0.807327151298523, -0.3503934442996979, -1.1988062858581543, -1.3775583505630493, -0.501115620136261, 0.5633214116096497, -1.210681438446045, 0.12358549237251282, 0.7495800852775574, -0.9154694080352783, -0.10830432176589966, -1.2867255210876465, 1.8765050172805786, -0.3852287530899048, -0.5488672256469727, 0.9106056094169617, -0.35632988810539246, 1.9573131799697876, -0.4492068290710449, -0.23337508738040924, 1.4352116584777832, 0.3301924467086792, 0.7918685078620911, 1.2238268852233887, -1.6734596490859985, 1.0385724306106567, -0.7598176598548889, 0.3231239914894104, 0.6635652184486389, 1.2081401348114014, 0.8319656252861023, -0.4891505837440491, -0.09946563094854355, -0.7116336822509766, -0.317050576210022, 0.8678260445594788, 0.6922799348831177, -0.15549437701702118, -0.30234208703041077, -0.2552598714828491, 0.049628064036369324, -0.5624845623970032, 0.9122341871261597, -0.2986432611942291, 0.21924325823783875, -0.30626943707466125, 0.997707188129425, 0.8974095582962036, 0.9386290311813354, 0.11198961734771729, -0.9746180772781372, 1.3662930727005005, -0.5299767255783081, 0.1555764228105545, -0.21674053370952606, -0.3897847831249237, 0.4863434433937073, 1.5968364477157593, 0.355329692363739, -1.1829171180725098, -0.31186357140541077, -0.6467381715774536, 0.695815920829773, 0.18757575750350952, -0.09163361042737961, -0.03859017789363861, 0.9643301367759705, 1.4111160039901733, -0.5052359700202942, 1.2019023895263672, 0.2856674790382385, 0.1677444726228714, 0.5057482719421387, 0.2186630517244339, -0.5067522525787354, 0.1905643343925476, -0.7574199438095093, -1.4356689453125, -0.6433265209197998, -0.15104736387729645, -0.6837127208709717, 2.2443313598632812, -0.07953702658414841, 0.3218388855457306, -2.102324962615967, -1.0432515144348145, -0.4550255537033081, 2.4686131477355957, 0.4497739374637604, 0.009969891048967838, -0.2320672571659088, -1.0193672180175781, 0.2604089379310608, 0.8362262845039368, -0.7636758685112, 0.6558315753936768, 0.17712298035621643, -0.9311532378196716, -0.6403496265411377, -0.31546705961227417, 0.5614141821861267, 1.028555154800415, -2.2905938625335693, -1.2193862199783325, 0.00844729132950306, -0.1301683932542801, 0.26508209109306335, -0.610893964767456, 0.6194338202476501, -0.37911057472229004, 0.7149343490600586, -0.8567208647727966, 2.2684648036956787, 1.6129781007766724, -1.5974386930465698, 1.2913331985473633, 0.7207849025726318, 0.5943710207939148, -0.23780381679534912, 0.945700466632843, -2.323439121246338, 2.1624181270599365, 0.21083232760429382, -1.0780400037765503, -0.0010399656603112817, -0.7516841292381287, 1.2908744812011719, 0.10858238488435745, -0.9137144088745117, 0.31297245621681213, -1.0271974802017212, -0.7308693528175354, -0.13109968602657318, 0.3271806240081787, 0.638152539730072, 0.6241794228553772, -1.7784852981567383, 0.11339972168207169, -0.39322444796562195, -0.808447003364563, -0.3636566698551178, 1.0010018348693848, 1.176578164100647, 0.029404448345303535, 0.3348907232284546, 0.3251042068004608, 0.37620115280151367, -0.12972640991210938, 0.21468575298786163, 0.7161930203437805, 0.6299254894256592, -0.9433961510658264, -1.3173733949661255, -1.196667194366455, 1.6173129081726074, -0.4686083495616913, 0.366809219121933, 1.694450855255127, -1.0453786849975586, 0.04516409710049629, 0.8159224390983582, -0.724560558795929, -1.297470211982727, 0.09251090884208679, 0.5367145538330078, 0.13112406432628632, -0.48240017890930176, -0.9271553158760071, -1.0250645875930786, 1.8433351516723633, -0.08612947911024094, -2.2082581520080566, -0.33063724637031555, 1.86093008518219, 0.9539127945899963, -1.3795663118362427, 0.06965398043394089, -0.4509682357311249, -0.3583778142929077, 1.6227351427078247, 0.08321196585893631, 1.5152091979980469, 1.4309375286102295, 0.46675965189933777, -1.0050081014633179, 0.36450475454330444, 0.7573446035385132, -0.4800563156604767, 0.321723073720932, 2.179112195968628, 0.6615631580352783, -1.018504023551941, 0.1009894460439682, 0.07038616389036179, 1.4370635747909546, 0.7396204471588135, -0.15722060203552246, -2.0896997451782227, -1.1415716409683228, -0.8266239166259766, -0.37922245264053345, -1.05547034740448, -0.13431258499622345, 1.5851655006408691, -1.9076818227767944, -1.8656760454177856, 0.5369816422462463, 0.6616964340209961, -1.857608437538147, 0.4835250675678253, -0.7773807048797607, 1.1572355031967163, -1.0711857080459595, 0.33233213424682617, 1.4322510957717896, -0.7971236705780029, -0.4222855269908905, 0.4825785756111145, -2.3023359775543213, -0.7301530241966248, 0.25981801748275757, 1.7355506420135498, -0.40296512842178345, -0.4769611060619354, -1.968615174293518, -0.3208543062210083, 0.7332196235656738, 0.6205493807792664, -0.030728965997695923, -0.11004018783569336, -1.83894681930542, 0.8187444806098938, -0.3832290768623352, 0.200384721159935, 0.7490919232368469, -0.4625709652900696, -1.214423656463623, -0.8171597719192505, -0.034449588507413864, -0.9375916719436646, -0.1267978399991989, 0.21683426201343536, -0.5619096159934998, 0.5020999312400818, -0.0922698974609375, -0.321537047624588, 0.5622417330741882, 0.7297956943511963, -0.30788055062294006, -1.2393240928649902, -0.4323700964450836, -2.2101383209228516, -1.0160930156707764, -1.491713047027588, -1.3827366828918457, -1.1196966171264648, -0.2858535647392273, 0.7068396210670471, -0.040887657552957535, -1.5583552122116089, -0.4000307321548462], [-1.7992408275604248, 0.6336855292320251, -1.1175458431243896, 0.41683316230773926, 0.7400457262992859, -0.855082631111145, -0.6397275924682617, -0.1122727021574974, -0.5085621476173401, -0.3051437735557556, 0.4982645511627197, -1.2133879661560059, 0.6076703071594238, 0.49245837330818176, 0.034216396510601044, 0.03403449431061745, -2.2479331493377686, -1.3765093088150024, -0.7888764142990112, 0.9894916415214539, -0.5143917202949524, -0.09844374656677246, 1.2890690565109253, 0.3145737946033478, -0.5889947414398193, -0.5744021534919739, 0.680340588092804, -0.3435092568397522, -0.4414826035499573, -0.42748475074768066, 0.004753139801323414, -0.022686056792736053, -1.4726619720458984, 0.4362213611602783, 0.932530403137207, 1.1855227947235107, 0.300610214471817, -0.914647102355957, -1.941945195198059, 1.1978245973587036, -0.044808488339185715, 0.22645807266235352, 0.5058602094650269, 0.7540509104728699, -0.504002571105957, 0.6402392387390137, -1.7195910215377808, 0.4610707759857178, -1.0543304681777954, -0.8277673721313477, -0.372721403837204, 0.7181453108787537, -1.461526870727539, -0.16385620832443237, 0.7511706352233887, -0.5628148317337036, 1.8987981081008911, 0.618594765663147, -0.5783621072769165, 1.35637629032135, -1.1695152521133423, -0.0676853358745575, -1.1300276517868042, 1.5158100128173828, 0.4746522307395935, 1.0574870109558105, -1.863480806350708, -0.9594829678535461, -0.7565041780471802, 0.08373194932937622, 0.14780399203300476, -0.5958805680274963, -1.408862829208374, 0.33789312839508057, 0.7041674256324768, -0.2128382921218872, -1.0364476442337036, 0.11610565334558487, -2.4464704990386963, 1.617438793182373, 0.5462415814399719, 1.2433691024780273, -0.02153848484158516, -1.6124283075332642, 0.2412683367729187, 1.1631412506103516, -0.4235774576663971, -0.2693803012371063, -0.7298232316970825, 0.41073286533355713, -0.2968437671661377, -0.829375147819519, 2.1329946517944336, -1.0351531505584717, 1.4394594430923462, 0.5089556574821472, 0.021014370024204254, 0.6177688241004944, -0.6810114979743958, 1.2186421155929565, -0.15856611728668213, 0.27681562304496765, -1.2165331840515137, -0.49031174182891846, 1.560874104499817, -1.5027695894241333, -1.769192099571228, 0.4616978168487549, 1.283200979232788, 0.7078384160995483, 1.232123851776123, 0.8633943796157837, 0.310134619474411, -0.365739107131958, 1.6187206506729126, -0.8081948161125183, -0.3306964635848999, 0.45019102096557617, -0.4205732047557831, -0.06973223388195038, 0.10996685177087784, 0.052415721118450165, -0.1679220199584961, 0.31312283873558044, 0.5209329128265381, -1.5543009042739868, -1.3396155834197998, 1.4249211549758911, -0.30134016275405884, -0.03903578966856003, -0.8914971947669983, -0.14276424050331116, 0.4322703778743744, 0.5474892854690552, 0.3164026737213135, -0.027071291580796242, -0.0335996188223362, -0.9336338639259338, -0.10781385749578476, 0.5383645296096802, -0.6296219825744629, 1.1220585107803345, -0.06352770328521729, 0.6422670483589172, 1.3106369972229004, 0.1532365381717682, -0.45298895239830017, 1.8044885396957397, -0.12258109450340271, 0.20790016651153564, -0.8418092727661133, -0.2233470231294632, -0.09344644844532013, 0.44587764143943787, -1.5015449523925781, -1.2084102630615234, -0.509518027305603, 1.3155264854431152, -0.9136859178543091, 0.4104974567890167, -1.2044216394424438, 0.1923898458480835, 1.2588565349578857, -0.564548134803772, -0.8299391865730286, -0.37950512766838074, 0.7508776187896729, 0.14898787438869476, -0.9045079350471497, -1.4786592721939087, 1.3365962505340576, -0.2369079738855362, -1.0025629997253418, -0.874919056892395, -0.1929587870836258, 0.7853439450263977, 0.14255733788013458, 1.4033535718917847, 2.64520263671875, 1.8921259641647339, -0.10830391943454742, -0.7098429203033447, 0.5267274975776672, -0.81859290599823, -1.1940635442733765, -0.49615541100502014, 0.3253461718559265, -0.34834712743759155, -0.9015970826148987, -1.099998950958252, -0.32318782806396484, -0.3744339644908905, -1.7261515855789185, 1.5677546262741089, 0.8365795612335205, 0.7423462271690369, 0.7334597110748291, -1.1434255838394165, 0.2548935115337372, -0.5073075890541077, 0.4088091254234314, 0.43008846044540405, 1.3383376598358154, -0.4929526448249817, 0.8243233561515808, 0.2233932912349701, 0.6887323260307312, 0.7456619739532471, 0.48274943232536316, -0.05633755400776863, -1.1096402406692505, -1.1135066747665405, 0.07500390708446503, -0.5839526057243347, -0.8324493169784546, 1.3905192613601685, -0.835528552532196, 0.04679442197084427, 0.7046358585357666, -0.853954017162323, -1.7121292352676392, -1.4498885869979858, -1.162671446800232, 1.2208107709884644, -0.2338605374097824, -0.9020105004310608, 1.4667593240737915, 0.036033473908901215, -2.098670244216919, 1.1359509229660034, -0.6627845168113708, 0.9308309555053711, 0.5973419547080994, -1.0197806358337402, -0.39481836557388306, 0.0003072099352721125, -0.44367894530296326, -1.1253618001937866, 0.7552692890167236, -0.22378911077976227, -0.2776073217391968, -1.0487406253814697, -0.19600361585617065, -1.1468557119369507, -0.8843593001365662, 1.0795098543167114, -0.09258557111024857, 0.019938930869102478, 1.3187731504440308, 0.6047413945198059, -1.2352186441421509, -0.5160769820213318, 0.9818949103355408, 0.26571646332740784, -0.7249745726585388, -0.9805247783660889, -1.4820778369903564, 0.4159451425075531, -0.09767021238803864, 0.24113184213638306, 0.7177504301071167, -0.4797273278236389, 2.2961137294769287, -1.3273472785949707, 1.1937583684921265, 0.6367756128311157, 1.4462758302688599, -0.13856667280197144, 0.6814826726913452, 0.3026355504989624, 0.831552267074585, -0.7641222476959229, 0.33261728286743164, -0.37237250804901123, 0.8697189092636108, -0.14413300156593323, 1.1469740867614746, 1.3639893531799316, 0.3269345164299011, 0.18855056166648865, 1.6710163354873657, -0.8994774222373962, -0.905825138092041, 0.009363323450088501, 0.7885271310806274, -1.0295628309249878, 1.1952614784240723, 1.4860519170761108, 2.430534839630127, -0.13085973262786865, 1.4372117519378662, -0.7534263730049133, 0.36963871121406555, 1.052947998046875, 0.07357075810432434, 0.27033549547195435, 0.7592614889144897, -0.023369064554572105, 0.5512493848800659, 0.1911381185054779, -0.1343739628791809, -0.9260401129722595, 0.0005736702005378902, 0.6000460982322693, 1.1720689535140991, 1.718658208847046, 1.9961284399032593, 0.7753622531890869, 0.02611817792057991, 1.9268120527267456, 0.9263246655464172, -0.3757520020008087, -0.3595791161060333, 1.0051119327545166, 1.454015851020813, 0.9363654255867004, -0.5758241415023804, -0.9409846663475037, 0.3966512978076935, -0.5229297876358032, 0.7991711497306824, 0.07796568423509598, 1.47702956199646, 0.15869836509227753, -0.021839531138539314, -0.9441710710525513, -0.601601243019104, -1.5075715780258179, 0.31508833169937134, 1.7795062065124512, 1.25269615650177, 0.2194564789533615, -0.2758393883705139, -1.5069084167480469, 1.4696577787399292, -0.965958297252655, 0.03452666476368904, 2.046579360961914, -0.8859478831291199, 1.6206750869750977, 0.1797153800725937, 1.2171553373336792, -1.9966671466827393, 0.2580353617668152, 1.21428382396698, -1.0515649318695068, 1.013088583946228, -1.4412258863449097, -0.06727030873298645, 0.05023939535021782, 1.1784820556640625, 1.4168412685394287, 0.38613829016685486, 0.6277468800544739, 0.29906097054481506, 0.15543285012245178, 0.5619442462921143, -0.6606974601745605, -1.799239993095398, 0.36795753240585327, -1.362463355064392, -1.5580179691314697, -0.47881051898002625, 0.06348787248134613, 2.1133904457092285, -0.8940222859382629, 0.4539645314216614, -0.8960977792739868, 0.3195718228816986, 0.24261631071567535, -1.755698800086975, -0.8474744558334351, 0.7070556879043579, -0.8848468661308289, -0.22098031640052795, -1.1514873504638672, -1.3894318342208862, -1.08989417552948, -0.5826398730278015, -2.3101141452789307, 0.3855474889278412, 0.12984336912631989, -1.3203165531158447, 0.4801926016807556, 0.5638256072998047, -1.1658694744110107, -1.1512993574142456, -1.0834460258483887, 0.00872834399342537, -0.12778832018375397, 0.943863570690155, -0.4611486792564392, -0.8112138509750366, -0.6418617367744446, -0.40449073910713196, 1.170397162437439, -0.07632729411125183, 1.0606508255004883, -0.6945741176605225, 1.0590217113494873, 0.591324508190155, -0.219233900308609, -0.18245261907577515, 0.14416702091693878, -0.09702136367559433, -1.0375878810882568, 1.711477518081665, 1.6602956056594849, -0.5174612998962402, 0.29743579030036926, 0.744584321975708, -0.2017713487148285, 0.15766632556915283, -1.1926031112670898, 0.1252928078174591, -1.3845462799072266, 1.4848666191101074, -1.1830335855484009, -0.8183285593986511, -1.1779859066009521, 0.8197864294052124, -1.4218398332595825, 1.5292198657989502, -0.7698395848274231, -0.9168894290924072, 0.3606574237346649, -2.456324577331543, 1.282549262046814, 1.441834807395935, -0.5148311257362366, -1.3401575088500977, -0.2589026689529419, 0.06063913181424141, 0.5215164422988892, -0.7821553349494934, -0.48949742317199707, 1.1724655628204346, 0.11475969851016998, -2.0501856803894043, -0.7267501950263977, -0.3719140887260437, -1.105354905128479, -1.529943585395813, 1.3399938344955444, -0.09670110791921616, 1.3542461395263672, -0.4327790439128876, -0.44812196493148804, -0.2407521903514862, 1.730691909790039, 1.0375264883041382, -1.034277081489563, 0.8028196692466736, -0.003043543780222535, -0.31325045228004456, 0.6889673471450806, -0.2214156687259674, 1.0109375715255737, 0.45192521810531616, -0.1430482715368271, -0.01349592860788107, -0.38637980818748474, 1.4024933576583862, -0.9007426500320435, -0.6550514101982117, -0.6044138073921204, -2.965787172317505, -0.1422530561685562, 0.265699565410614, -1.6061941385269165, -0.44195184111595154, -0.9339867830276489, -0.5778188109397888, -2.3523507118225098, -1.1486440896987915, -0.6811478734016418, 0.3665948212146759, 0.6361807584762573, 1.4052425622940063, -0.19619792699813843, 2.258293867111206, 0.17043156921863556, 0.5831817388534546, -0.6296765804290771, 0.9864898920059204, -1.5748978853225708, 0.9611122608184814, -0.6837133765220642, -1.1733245849609375, 0.8049459457397461, 1.1361504793167114, -0.319280743598938, -0.3225244879722595, -1.6874723434448242, 1.2646925449371338, -1.8151112794876099, -0.6470667123794556, -1.6219531297683716, -2.1145408153533936, -0.05233887955546379, 0.26628217101097107, 0.7044485211372375, 1.0895191431045532, -0.6004061698913574, 1.2480415105819702, -1.0497715473175049, -1.7506811618804932, 0.004761955700814724, -0.52754807472229, -0.0370059609413147, 0.5776951313018799, 0.21505418419837952, -1.3779592514038086, -0.8344358205795288, -0.21018795669078827, 0.025067675858736038, 1.1016614437103271, 0.7589171528816223, 0.9317578077316284, -0.5667921900749207, -1.4816161394119263, 0.7970032691955566, 0.3685082793235779, 0.2166348695755005, 0.09636793285608292, -1.2273119688034058, 1.313036322593689, 1.2729177474975586, -0.8687161207199097, -1.2817939519882202, -0.49794071912765503, 0.1486782729625702, -0.39309999346733093, 1.471665859222412, 1.4735329151153564, -1.5766088962554932, -0.97134929895401, -0.4722817838191986, 0.5908008217811584, -0.1590249091386795, -1.4065635204315186, 0.14172399044036865, 0.48890233039855957, -0.9773879647254944, -0.41393616795539856, -0.6506272554397583, -1.2706736326217651, -2.2074368000030518, -0.23506656289100647, 0.6064234972000122, -0.022790536284446716, -1.404276967048645, 0.5677065253257751, 0.26634687185287476, 0.6753032803535461, 1.4025503396987915, 0.04668877646327019, 1.2429088354110718, -0.6475653648376465, -1.5791372060775757, 0.5488048791885376, -0.283676415681839, -0.7757222056388855, 1.4099997282028198, -0.6147942543029785, 0.7921703457832336, 0.05453425273299217, -0.5000431537628174, -0.38127240538597107, 1.4435701370239258, -2.7501797676086426, 0.36432817578315735, 1.6699886322021484, -0.2813800573348999, -0.42207300662994385, -0.5676153302192688, 1.4117541313171387, 0.09621167927980423, -0.3662228286266327, -0.8504729270935059, 2.673142910003662, 1.3721551895141602, 0.3391847610473633, -0.989485502243042, 0.3488093912601471, -0.8035715222358704, -0.5695887207984924, 1.0476396083831787, -0.9071540236473083, -1.904584288597107, 0.8979916572570801, -0.4172058701515198, 0.32222622632980347, -2.3896872997283936, 1.6368266344070435, 0.26426056027412415, 1.0768393278121948, 0.6046473383903503, 0.363142728805542, 1.032608151435852, 0.5170049667358398, 0.2012859433889389, 1.228080153465271, 0.9233465194702148, 0.7418965101242065, -2.386322259902954, -1.7934778928756714, -0.14941753447055817, -0.7126627564430237, -1.2264479398727417, 0.34929922223091125, 1.8194299936294556, 0.5253305435180664, -0.08653774112462997, 0.6436464786529541, 0.14079411327838898, 0.07716682553291321, -2.370296001434326, -0.5465652942657471, 1.1949580907821655, -1.479958176612854, 1.7781931161880493, 0.05514812469482422, 1.2146039009094238, -0.00484063196927309, -0.7760951519012451, 0.8894997239112854, 1.1351475715637207, -1.1379592418670654, -0.11284525692462921, -0.09366374462842941, -0.43361830711364746, -1.4868412017822266, -0.3626891076564789, -0.35569989681243896, 0.944531261920929, -0.7417367100715637, -0.7782586216926575, 0.5596305727958679, -0.7956962585449219, -0.6449232697486877, -0.10687711834907532, -1.0475704669952393, 0.3471470773220062, 1.8551299571990967, 1.022082805633545, -0.6827720403671265, 0.0701829120516777, 0.0676710233092308, 1.6156870126724243, -0.014061210677027702, -0.5526912808418274, -0.6237377524375916, 1.1192874908447266, 1.5769540071487427, 1.397044062614441, -0.43992048501968384, 0.8784056305885315, -1.4865715503692627, 0.6185539960861206, -0.5964195728302002, -0.9931484460830688, -0.13483202457427979, -0.567385733127594, -0.39002785086631775, -0.43819859623908997, 0.4195040166378021, -0.19263993203639984, 0.44823992252349854, -0.20836381614208221, 1.2461998462677002, 1.031589150428772, -0.3456529676914215, 0.4868427515029907, 0.5349037647247314, -0.5960767865180969, -0.5670053958892822, -0.3192892372608185, 1.5131856203079224, -0.6975149512290955, -0.09683039784431458, -1.4062349796295166, -0.07356466352939606, 0.45749804377555847, -2.1676199436187744, -1.0572232007980347, 0.1640876978635788, -0.43498852849006653, -0.30082568526268005, -1.668176531791687, -0.000458975147921592, -0.29552578926086426, 1.254888892173767, 0.10967271029949188, -0.8236470818519592, 1.0685352087020874, 0.04620327800512314, -0.8455116748809814, -1.2347328662872314, -0.47849932312965393, -0.2379806786775589, 0.7298632264137268, -0.8737288117408752, -0.1242619976401329, 0.5501748919487, -0.6463646292686462, 0.34631651639938354, 0.40724247694015503, -0.9423282742500305, 1.247728943824768, 1.7034953832626343, -1.4727764129638672, 1.1751880645751953, 0.05715706944465637, 1.0614668130874634, 2.2969396114349365, 1.4137437343597412, 2.313361167907715, -1.2649667263031006, 0.06809951364994049, 0.8681362271308899, -0.3832319676876068, -0.31807148456573486, -1.5715876817703247, 1.9677579402923584, 0.5687660574913025, 0.2953222692012787, 1.157628059387207, 0.12016253918409348, 0.42215874791145325, -1.1981884241104126, 2.0589730739593506, 1.204930067062378, 0.22225798666477203, -0.508945882320404, 0.696610689163208, -2.0519182682037354, -0.19637304544448853, -1.4887597560882568, 0.6378533840179443, 1.1667474508285522, 0.6703546047210693, -0.879389762878418, -0.8737802505493164, -0.679184079170227, 1.1162256002426147, -1.9387651681900024, -0.5837222933769226, 0.38589632511138916, -0.2985019087791443, 1.1226117610931396, -0.582688570022583, -0.9240303039550781, -0.2851294279098511, 0.7414653897285461, -1.5307857990264893, -0.7975255250930786, -1.1877142190933228, 0.6897751092910767, 1.8500463962554932, -0.9812127351760864, 0.2616146206855774, -0.8912147283554077, 1.1492000818252563, -0.710793137550354, -1.7589507102966309, -0.6755549311637878, 0.690395176410675, 0.6570070385932922, 0.9161185026168823, -2.0772035121917725, -1.477195382118225, 2.1869313716888428, -0.26142293214797974, -0.24511051177978516, -0.8717895746231079, -0.6588001251220703, 2.290518045425415, 0.09736479073762894, -1.0922996997833252, -0.6472580432891846, -0.05327516794204712, 0.17020438611507416, -1.4129194021224976, -0.023473139852285385, 0.35426491498947144, 0.4236774742603302, 0.15518435835838318, 1.0148156881332397, -1.095373272895813, 0.7800394892692566, -1.8556729555130005, 1.4446170330047607, -0.8092586398124695, 0.7635740041732788, -0.22065666317939758, -0.2164492905139923, 0.2803462743759155, -0.9532260298728943, 2.387430429458618, -0.257110059261322, 0.9259656667709351, 0.06532282382249832, 0.719788134098053, 0.370849609375, 1.446083664894104, -1.592393159866333, -0.15935677289962769, -0.9678202271461487, -1.000807762145996, -1.5184321403503418, -0.7081740498542786, 0.6267452239990234, -0.005569361615926027, 0.7824184894561768, 1.0287408828735352, 0.6603057384490967, -1.0441399812698364, -0.13607294857501984, -1.0411534309387207, 0.4821731150150299, 0.5200746059417725, 0.06480883061885834, 0.28892046213150024, -0.13062027096748352, 0.03196628764271736, 1.58217453956604, -0.7167472243309021, -0.6336066722869873, -1.9793589115142822, 0.19892753660678864, 0.6652745604515076, -0.5110624432563782, -1.1960151195526123, -0.032420381903648376, 1.43618905544281, 1.0752062797546387, 0.09169948846101761, -0.3203657567501068, -0.808517336845398, -0.7232618927955627, -1.1044179201126099, -0.0910620465874672, -0.545332133769989, -1.3572173118591309, -1.5812009572982788, 1.0983237028121948, 0.7711886763572693, -1.4252605438232422, 1.5977941751480103, 1.349442481994629, 0.5095518827438354, -2.100863456726074, 0.5416331887245178, -1.0337668657302856, -1.1358505487442017, 0.02491198293864727, 0.14105114340782166, -0.12224830687046051, -0.0012568766251206398, 0.2072572112083435, -0.7157158851623535, -0.19799941778182983, -0.30945470929145813, -1.3112586736679077, 0.9443464875221252, -0.784905195236206, -0.6763861775398254, -0.6911396980285645, -0.9966574311256409, 0.6739523410797119, 1.4646083116531372, 1.9590812921524048, -0.6284533739089966, 0.8860633373260498, 0.21905024349689484, 0.22683075070381165, 3.0587966442108154, 0.40938490629196167, -0.6161206960678101, 1.0269838571548462, -1.7024002075195312, 0.2409083992242813, 0.5154826641082764, -0.20026154816150665, -1.7439234256744385, 0.6254193782806396, -1.509785771369934, -0.032564498484134674, -0.09055172652006149, 1.259173035621643, 0.9223020672798157, 0.07102438807487488, -0.47844961285591125, 0.24296444654464722, 0.2553733289241791, 0.7702766060829163, 1.501721978187561, -1.1523483991622925, -0.027226237580180168, 0.5958324074745178, -0.6048616766929626, 0.32981348037719727, 1.105239748954773, 0.4239633083343506, 0.17056798934936523, -1.2553893327713013, -0.6527519226074219, -1.8248934745788574, 0.08455076813697815, 0.9624837636947632, 0.3720950782299042, -1.7296396493911743, -0.32849135994911194, -0.5841693878173828, 0.5723276138305664, 0.8398045897483826, 0.5315430164337158, 1.6749504804611206, -0.6417040228843689, 0.6818268299102783, -0.5820322036743164, -1.1652132272720337, -1.1479586362838745, -1.2127561569213867, -1.7169206142425537, 0.24525126814842224, -2.1156187057495117, -0.18526479601860046, 0.34315043687820435, 0.5050815343856812, 0.21198461949825287, 1.447424054145813, 1.5104365348815918, 0.7531559467315674, 1.0722354650497437, 0.15187060832977295, -0.7026486396789551, -0.6220040321350098, -0.35175010561943054, 0.6839203834533691, -0.377663791179657, 0.9076125025749207, -1.1858749389648438, 0.04498962685465813, -2.753988742828369, 1.4349970817565918, -0.174162358045578, 0.2820028066635132, -0.9725102186203003, -0.3091674745082855, -0.8266023993492126, -0.578948438167572, -0.14395694434642792, 0.07141014188528061, -1.2819300889968872, 0.6258432865142822, -0.4844743311405182, -1.3318244218826294, -0.45296168327331543, 0.0855904221534729, -1.1587951183319092, -0.11720278114080429, 0.9840443134307861, -1.1897540092468262, -2.1485555171966553, 1.5094202756881714, -0.9208453297615051, -0.4477218687534332, -0.20173558592796326, 1.4896748065948486, -1.1195920705795288, -0.5687685012817383, -0.13645218312740326, -1.4462130069732666, -1.8511146306991577, -0.36870014667510986, 1.6510268449783325, 0.455282598733902, 0.2537141442298889, -1.8237793445587158, -0.28105735778808594, 1.0266380310058594, 0.11427655071020126, -0.2028064876794815, 1.2649588584899902, 0.6088080406188965, -0.25935932993888855, -0.12345512956380844, -0.6377261281013489, -0.18019695580005646, -0.944525420665741, 0.06775163859128952, -0.9148144125938416, -0.07859399914741516, 0.39272186160087585, 2.5728397369384766, -0.7423710227012634, -1.5418983697891235, 0.5014241933822632], [-0.03288714960217476, -0.2959301769733429, -1.9142142534255981, -0.832033097743988, 0.026849884539842606, 1.4985235929489136, 1.2214298248291016, 1.5904961824417114, -0.2638893127441406, 0.0784534364938736, 0.4277071952819824, -1.8230239152908325, 0.6859414577484131, -0.29813292622566223, -0.4363957345485687, -1.133133053779602, 0.020043082535266876, -0.5786054134368896, -0.016476118937134743, -0.019360175356268883, -0.520577073097229, -0.8278453946113586, -0.3980923891067505, 0.5567185878753662, -2.1517765522003174, 0.7547720670700073, -0.11824537813663483, 0.8706782460212708, -0.015204190276563168, 0.5367852449417114, -0.23117497563362122, -0.842218816280365, -0.39085355401039124, 0.22038055956363678, -0.8974083662033081, -0.13468120992183685, 0.12974898517131805, 0.830653965473175, -0.5473921298980713, 0.39844152331352234, -0.38508206605911255, -0.3624234199523926, 1.014998197555542, -1.0505597591400146, 0.25923657417297363, -0.9034250378608704, -0.39077356457710266, -1.7051634788513184, -0.30266374349594116, 1.066623330116272, 0.7198996543884277, -0.988960862159729, 0.2221318930387497, 1.1184676885604858, 0.6470590233802795, 2.213862895965576, -0.6432800889015198, 0.5349859595298767, -0.19933576881885529, -1.1601670980453491, 0.22403663396835327, -1.1326453685760498, 1.323266863822937, 0.6245490908622742, 0.5303438901901245, 1.8078162670135498, 0.20035015046596527, 0.5814655423164368, 0.31551089882850647, 0.7413211464881897, -0.018698111176490784, -0.6754392385482788, -0.5583998560905457, -0.13292792439460754, 0.4769982695579529, 2.008507251739502, 0.37318944931030273, 0.8597502112388611, -0.4169432818889618, 1.0456242561340332, 0.1897633820772171, 0.8891705870628357, -1.9462822675704956, -2.523163318634033, 0.11619123816490173, 0.12354470789432526, -1.3114572763442993, 0.5682799220085144, 1.9036258459091187, 0.5176314115524292, -0.6685152053833008, -1.3683249950408936, -0.8217818140983582, 0.7991182804107666, -1.1674437522888184, -1.433782696723938, -1.2471007108688354, 0.8717393279075623, 0.5210582613945007, 0.4172397553920746, 1.2085100412368774, -0.7273871898651123, -0.5784769058227539, 0.2366105318069458, 0.09593712538480759, -1.0510036945343018, -0.15827898681163788, 1.8879904747009277, 0.2221059799194336, 1.5126410722732544, 0.49800431728363037, 1.0395203828811646, 0.8183814287185669, -0.6867695450782776, 0.5096361041069031, 1.999796986579895, 1.2792034149169922, -0.08646364510059357, 0.33606454730033875, -0.24338598549365997, 0.12071279436349869, 1.2948634624481201, -0.6525877118110657, 0.33626630902290344, 0.527277410030365, -1.2148746252059937, -1.1537208557128906, -2.1078379154205322, -0.5202080011367798, -1.1279139518737793, 0.12255380302667618, -0.6690905690193176, -0.48940327763557434, 0.02605731040239334, -0.572609007358551, 0.9395753741264343, 0.19252265989780426, 0.38818714022636414, -0.06521797925233841, -2.239748954772949, 0.42876866459846497, -0.9053871035575867, -0.9428683519363403, -0.036646101623773575, 0.45562058687210083, -0.32217276096343994, -0.3739846646785736, 1.6903140544891357, 0.5135442614555359, 0.7816714644432068, 0.39514651894569397, -1.500455379486084, 0.563101589679718, 0.06209986284375191, -1.2856961488723755, -0.4904419183731079, -0.5573212504386902, 1.2616143226623535, -1.6844191551208496, 1.5000869035720825, 1.1398876905441284, 0.49019119143486023, -1.117518663406372, -0.057225827127695084, 0.33495673537254333, 0.53956139087677, 0.5344049334526062, -0.0996021032333374, 0.43349555134773254, 0.7586717009544373, -1.0914417505264282, 0.3825148046016693, -0.8718600869178772, -0.0619453601539135, 1.027087926864624, -1.277244210243225, -0.24113011360168457, -2.0622119903564453, -0.437004953622818, 1.9374125003814697, -1.6124937534332275, -0.09913807362318039, 1.1107773780822754, -0.44982069730758667, -1.079420804977417, -0.24268434941768646, -0.5094570517539978, 1.1207518577575684, 0.0596151277422905, 0.23500046133995056, 0.6206182837486267, -0.32020828127861023, -1.2374458312988281, 0.2899773120880127, 0.8982537984848022, 0.3241865038871765, 0.7181901931762695, 0.5491349101066589, -0.1584189236164093, 0.9167441129684448, 0.7496577501296997, 1.6103023290634155, 0.10303159058094025, -2.008854627609253, -0.12480911612510681, 0.3446751534938812, -1.1525362730026245, -1.0200241804122925, -1.7932701110839844, 0.16569827497005463, -0.11763256788253784, 1.0672860145568848, -1.850974440574646, 2.0329558849334717, 1.5039178133010864, 0.9684886932373047, 0.5828462243080139, 0.3234090209007263, -0.8303031325340271, 0.15351352095603943, 1.4822577238082886, -0.6567175984382629, -0.11475234478712082, 0.5309196710586548, -1.1282094717025757, -0.14859449863433838, 0.5810374617576599, -0.3250628709793091, 1.185263991355896, 0.5618388056755066, -0.6651855707168579, -0.09520253539085388, -0.4184381067752838, 1.2491823434829712, -0.5082768797874451, -0.2883293032646179, -0.8620936870574951, -0.29411178827285767, -0.324470192193985, -0.1922421008348465, -0.12465649098157883, 1.8212794065475464, 0.6656845808029175, 0.5944756269454956, 2.8422892093658447, -0.17572331428527832, 2.1167447566986084, -0.01885306090116501, 0.6543919444084167, -1.1523102521896362, -0.7224080562591553, -2.1447832584381104, -1.1268103122711182, 0.07199711352586746, -1.5498758554458618, -1.195480465888977, 0.17905065417289734, 0.08736859261989594, 0.16919051110744476, 0.3913719058036804, -0.21279063820838928, 1.753532886505127, 0.3480229675769806, -0.12727737426757812, -1.8523138761520386, 0.23029668629169464, -1.0756324529647827, 0.9494770765304565, -0.4784581959247589, -1.520628809928894, -1.1594313383102417, 0.9335576891899109, -0.29361289739608765, 1.5504049062728882, -1.2826638221740723, 0.8764474391937256, 0.7023751735687256, -0.9841339588165283, 0.006495485547930002, 0.9084365367889404, -0.7676500678062439, -0.47864973545074463, 0.46147194504737854, 1.5925410985946655, -0.6751531958580017, -0.46705859899520874, 0.01286875456571579, 0.4869473874568939, -0.2716308832168579, 1.1033861637115479, 1.1087673902511597, 1.114685297012329, -0.07804755866527557, 1.365169644355774, 0.7467129826545715, 0.5089071989059448, 0.207443967461586, 0.22535809874534607, -1.0582101345062256, -0.1457633078098297, -1.4280225038528442, -0.055066950619220734, 0.5609579682350159, 0.5890226364135742, 0.8414255380630493, 0.8135776519775391, 0.7838574051856995, -0.4182904064655304, 1.308660626411438, -1.3077237606048584, -0.07339999824762344, 0.06380750238895416, 0.34660717844963074, 0.7549297213554382, 1.174274206161499, -1.9007830619812012, 1.2946257591247559, -1.977183222770691, 1.5501261949539185, -0.2871038019657135, -0.9229021072387695, 0.34027838706970215, -1.1838699579238892, -0.07797402888536453, -0.2710890471935272, 0.2417343109846115, -1.9778589010238647, -1.371053695678711, -0.411415159702301, -0.5392288565635681, 1.4046027660369873, 1.7865818738937378, 0.24369092285633087, 0.5037996172904968, -1.2557789087295532, -1.1436375379562378, -1.9780831336975098, 0.5450609922409058, 1.7173422574996948, 1.2347034215927124, -0.6159545183181763, 0.3540092408657074, -0.6009626984596252, 0.7598584294319153, -0.9144670367240906, -0.6151651740074158, 0.5577660202980042, -0.9969502687454224, 2.249241828918457, 0.7632557153701782, -1.737730860710144, 0.06706555932760239, -0.5042831897735596, 0.6193366050720215, -0.6329454183578491, -0.70931476354599, 1.93893563747406, 0.3208479583263397, 2.1788856983184814, -0.8670551180839539, -1.2238703966140747, 0.49204468727111816, 0.2427329123020172, -0.32991334795951843, -0.3736391067504883, 0.4898785352706909, 1.3165422677993774, -1.386748194694519, 0.8956311941146851, 0.28401222825050354, 0.0451783649623394, 0.8515340089797974, 1.6301835775375366, -0.01579052209854126, 0.9867995977401733, 1.2188514471054077, 0.035184748470783234, -0.7356388568878174, 1.532140851020813, 0.03554004430770874, 0.8943925499916077, -0.2762586176395416, 0.03678886964917183, 1.8330012559890747, -2.0135772228240967, 0.6481293439865112, -0.2342662364244461, -0.6570871472358704, -0.9541114568710327, -0.8390592336654663, 0.5394046306610107, 0.08970267325639725, 1.703292727470398, 0.4248422086238861, 0.951612114906311, 0.27220726013183594, -0.18381193280220032, 1.0307568311691284, 0.7769407033920288, -0.15825076401233673, -0.2867153286933899, -0.7564301490783691, -0.8651493191719055, 0.6936690211296082, 0.8293517827987671, 2.169877290725708, -0.41299694776535034, 0.1267876923084259, -0.15904749929904938, 0.33235448598861694, 0.4286612868309021, -0.009060355834662914, 0.6712833046913147, -0.2115131914615631, 0.5477131009101868, 1.558415412902832, -1.1523274183273315, -0.02523081935942173, -0.06234751641750336, -0.7221498489379883, 1.4487560987472534, -0.8015563488006592, -0.34417474269866943, 0.06616346538066864, 0.19833262264728546, -0.01577947847545147, -0.8181403875350952, -0.260562539100647, -0.5409128069877625, -0.33629393577575684, 0.29500776529312134, 2.2121598720550537, 0.30216747522354126, -0.5222676396369934, -1.5160176753997803, -0.9819807410240173, -0.5277012586593628, -2.047926664352417, -0.6484479308128357, 0.341968297958374, 1.9028546810150146, 0.09874892234802246, 1.6787290573120117, 0.4914018511772156, 0.8131659626960754, 2.299276351928711, -0.1280088722705841, -0.8605544567108154, 0.5294193029403687, -0.16406363248825073, -1.8608397245407104, -1.3579494953155518, -0.7501096725463867, 0.8638150095939636, 1.203923225402832, -1.80715012550354, 0.34099119901657104, 0.5191522836685181, 2.124746084213257, 0.9069144129753113, 0.2906792163848877, 1.2607094049453735, -2.046879768371582, 0.14804697036743164, 2.3538057804107666, 0.5816148519515991, 0.9718465805053711, -0.8186111450195312, -0.660651445388794, 1.029853105545044, -0.5830193758010864, -0.36546123027801514, 0.3396683633327484, -0.5397685170173645, -1.8743116855621338, 1.239729881286621, -3.663435220718384, 1.1555026769638062, -0.43229252099990845, -0.6883644461631775, 0.8705728650093079, 0.16297267377376556, -0.23148246109485626, -0.948590874671936, 0.18502087891101837, 1.1013481616973877, -0.5747518539428711, -1.0721733570098877, 0.9773319959640503, -0.3652936816215515, 3.3996639251708984, -1.1316337585449219, 0.508460521697998, 0.6611056923866272, 0.0013235188089311123, 0.8008820414543152, 2.128809928894043, 2.483949899673462, -1.023626685142517, 0.48380088806152344, 0.041581425815820694, -1.9600236415863037, -1.1221450567245483, 1.6412330865859985, 0.26138943433761597, 0.8588553667068481, -0.6914251446723938, 0.283282995223999, 0.21826224029064178, 1.1340646743774414, -0.7049616575241089, 0.6078200340270996, 0.19648806750774384, 1.2294880151748657, 0.029259633272886276, 2.4715991020202637, -0.7950618863105774, -1.732305884361267, -1.1655353307724, 0.20365767180919647, 0.312979131937027, 0.5243892073631287, 0.4012487232685089, -1.1418766975402832, 0.6708371043205261, -0.34869909286499023, -1.7725909948349, 0.790919840335846, 1.6646353006362915, 1.0603452920913696, 0.1304612010717392, 0.5481629371643066, 0.4103809595108032, -0.4533599615097046, -0.08358094096183777, 1.2887033224105835, 1.1955575942993164, 1.2238489389419556, -1.284288763999939, -0.6120355129241943, -0.17603859305381775, 1.5095386505126953, -0.649208128452301, -0.3947999179363251, 0.0969279333949089, 1.0631215572357178, 0.8818715810775757, 0.7721918225288391, -0.3042708933353424, 1.8168004751205444, -0.7713615894317627, 2.0644593238830566, 0.24745866656303406, -0.43014541268348694, 0.05282875522971153, 0.7315551042556763, -0.41808751225471497, 0.5146359801292419, -0.671408474445343, 0.8301615715026855, -0.465588241815567, -0.4154522716999054, -0.857005774974823, -1.1774020195007324, 0.8856713175773621, -0.8623563051223755, -0.6882859468460083, -0.3399133086204529, 0.516974925994873, -1.2885806560516357, -0.9749781489372253, -0.35919398069381714, -1.1070668697357178, -1.3756531476974487, -0.2787352204322815, 0.2344275265932083, -0.635532021522522, 0.6422123908996582, -0.8546015024185181, 0.3807829022407532, 0.7942524552345276, -2.154228687286377, -0.39250320196151733, -0.7084792852401733, 1.3533718585968018, 1.0971474647521973, 0.0776679739356041, -0.9759072065353394, -0.3851809799671173, -0.4757312834262848, 0.10016129910945892, -0.8007164597511292, -1.229821801185608, -0.17455017566680908, -0.6310304999351501, 0.46245765686035156, 0.5970364809036255, -0.37063395977020264, 0.9184010624885559, -0.0275510773062706, 0.8618541359901428, -0.12172411382198334, -2.1358275413513184, -0.7226569652557373, -0.8402600288391113, -0.886627733707428, 0.29708459973335266, 0.28307226300239563, -0.9303833842277527, 0.35890284180641174, -0.25648173689842224, -0.5737788677215576, 0.828805685043335, 0.6395288109779358, 1.2217557430267334, -0.14922116696834564, -0.7414480447769165, -0.9811001420021057, -0.014055522158741951, 1.619864583015442, 1.2397527694702148, 0.8513826727867126, -0.9726642966270447, -1.3764159679412842, 1.4871065616607666, -0.05138777941465378, -0.7458540797233582, -0.16480663418769836, -0.9356511235237122, 0.3204785883426666, 1.050655484199524, -0.8598148226737976, -1.2851848602294922, -0.6608229875564575, 0.2160687893629074, -0.09296058863401413, -0.18858622014522552, -1.4073582887649536, -0.21296539902687073, -2.0651795864105225, 0.9450510740280151, 0.6690006852149963, 1.577032208442688, 0.7459102272987366, 0.5467342734336853, -0.25018006563186646, -0.9697917699813843, -1.0597736835479736, 0.0918274000287056, -1.2483315467834473, -0.8108563423156738, 0.4405950605869293, 0.38207346200942993, 1.6296007633209229, -0.3027873933315277, 0.5740460753440857, 0.07747811824083328, 1.7581263780593872, -1.921812891960144, 1.4107881784439087, 0.38236507773399353, 1.141900897026062, -0.5289482474327087, 0.766136884689331, 0.37709444761276245, 0.3031109571456909, -0.7322115302085876, 0.18234588205814362, -0.35768988728523254, 0.1558942198753357, 1.1289700269699097, 1.2037386894226074, 0.0425855927169323, -0.06842000782489777, -2.2902262210845947, 1.3018947839736938, -0.6089770793914795, -0.6324528455734253, -1.804609775543213, -1.3521212339401245, -0.09222287684679031, -0.5440793037414551, 0.22857649624347687, 1.7375819683074951, 0.7587169408798218, 0.3481391966342926, -1.3205646276474, -0.8801670074462891, 0.3344459533691406, -0.9628199338912964, 1.2286393642425537, -0.14544886350631714, 1.1567082405090332, 0.9523298740386963, -0.41019734740257263, -1.7457951307296753, 1.2055917978286743, 0.8433561325073242, -1.5275870561599731, -0.1557529866695404, -0.18364034593105316, 0.31861644983291626, -0.06806206703186035, 0.13581469655036926, 0.5929733514785767, -1.7860777378082275, -1.0670503377914429, 0.3945521116256714, -1.6035150289535522, 0.636105477809906, 0.021517718210816383, 1.7124637365341187, 1.0039132833480835, 1.563717007637024, -1.0198078155517578, 0.38140395283699036, -1.4613687992095947, 0.10911928862333298, -3.488563060760498, -1.6254996061325073, -0.5959121584892273, 0.508756697177887, 0.6690534353256226, -2.152308702468872, 0.13368633389472961, 0.6325392723083496, 0.12686502933502197, 0.5968279838562012, 0.912693202495575, 0.5725497007369995, 0.04142335429787636, 0.23648175597190857, 1.2706267833709717, 0.562926709651947, -0.6327964067459106, -0.015374260023236275, 0.3157770335674286, 0.1461641490459442, -0.21682609617710114, -1.221437692642212, 0.48749715089797974, 0.76256263256073, 0.9322375059127808, 1.2034687995910645, 1.408745527267456, -0.32234132289886475, -0.14815132319927216, -1.3029183149337769, -1.2412039041519165, 2.084613561630249, 0.1796063929796219, -1.602668285369873, -0.8511278629302979, 0.8659405708312988, 0.3791755139827728, -0.4779432415962219, -0.3489331007003784, -0.44461360573768616, -0.2760752737522125, 1.4627152681350708, -0.6347686052322388, 0.34619906544685364, 0.716335654258728, -0.21330954134464264, 2.1760168075561523, 0.21009056270122528, 0.9788104891777039, -1.7095001935958862, 1.4323734045028687, -0.20267046988010406, -0.03761022537946701, 0.6805681586265564, 0.14448879659175873, -1.1326093673706055, -0.8570088148117065, -1.7430427074432373, -0.9307659268379211, -1.6238435506820679, -0.9156660437583923, 0.5220640897750854, -0.7020201683044434, 0.318633109331131, 1.9608442783355713, 0.29583972692489624, 0.22555623948574066, -1.1500303745269775, 0.08767686784267426, -0.5994256734848022, 0.39422571659088135, -0.6124434471130371, 1.0305272340774536, 1.0610038042068481, 1.0346981287002563, -1.8334827423095703, 0.5696923732757568, 0.6018456816673279, -1.3805512189865112, 0.21005116403102875, 0.25543537735939026, 0.21435213088989258, -0.41551336646080017, -0.6962095499038696, -0.5378161072731018, 0.7008653879165649, -0.5685720443725586, -0.33076024055480957, 0.9952201843261719, 0.7932456135749817, -0.2328716218471527, 0.896069347858429, 2.013335704803467, -0.14244329929351807, -0.39908313751220703, -0.3088364899158478, -0.05215312913060188, -1.0264045000076294, 1.456544280052185, -0.11727162450551987, 0.5219037532806396, 1.4454227685928345, 0.6019774079322815, 0.26832330226898193, -0.7854686975479126, -0.7664692997932434, -1.1062865257263184, 0.42996323108673096, 1.358734130859375, -0.45738765597343445, -1.2248201370239258, -1.0463050603866577, 1.2744861841201782, 0.6783187985420227, -1.37901771068573, -1.581894040107727, -1.0101503133773804, -0.262798011302948, -0.3572462797164917, 1.042965292930603, 0.8956244587898254, 1.7192275524139404, 0.42691612243652344, -0.5307531952857971, 0.882830023765564, 0.6204129457473755, -1.5698859691619873, -0.1654737889766693, 0.26393553614616394, 0.6968348622322083, 0.7385796308517456, -0.889168381690979, -0.11698479950428009, 0.6256411075592041, -1.4951400756835938, -1.4027490615844727, 0.22052061557769775, -1.1899778842926025, 0.6444151997566223, 0.02241654135286808, -0.34852612018585205, -0.1669793426990509, 1.4652652740478516, 1.8969193696975708, 0.6591615080833435, 0.771393895149231, 0.6700373291969299, 1.9369347095489502, 0.7926595211029053, -1.067134976387024, 0.1817464679479599, -0.5210910439491272, -1.4327400922775269, -0.1103207990527153, -0.09069942682981491, 1.4996672868728638, -0.09027441591024399, -0.215791717171669, 0.05309798941016197, 0.08758388459682465, 1.2407463788986206, 0.1356482356786728, -0.76301109790802, -0.5591439604759216, -0.16377957165241241, 0.3810434639453888, -1.7346820831298828, -0.36055296659469604, 0.23808975517749786, -0.21732325851917267, 1.2588859796524048, -0.4991762042045593, -0.49094894528388977, 0.6273399591445923, 0.5022244453430176, 0.10894551873207092, -0.5127748250961304, 0.4034525752067566, 0.6714978814125061, -1.1342519521713257, -0.26729342341423035, -0.11188904941082001, 0.5995451211929321, -0.18150575459003448, -1.3376307487487793, -0.6490969061851501, 0.21938006579875946, -0.5712565183639526, 0.046412721276283264, 1.2068997621536255, -0.41919195652008057, 0.483417809009552, 0.03373179957270622, 0.34247884154319763, 1.2756950855255127, -0.3834768533706665, -0.16733543574810028, -1.573664903640747, 2.245054006576538, 0.4908570349216461, -1.5377954244613647, 0.5710841417312622, -0.14264298975467682, -2.005896806716919, 0.4827911853790283, -0.466007262468338, 0.19389860332012177, -0.38412532210350037, -0.10680986940860748, 0.6838629245758057, 1.1147089004516602, -0.5378730893135071, -0.6232845187187195, -0.3748764097690582, -0.18803371489048004, 0.6101462244987488, 1.2773587703704834, -2.2096502780914307, -1.481091856956482, 0.9069616198539734, 0.6458537578582764, 1.762351632118225, -0.2815488278865814, 0.7239090204238892, -2.107067108154297, 1.0940823554992676, 0.5634880065917969, -0.8541127443313599, -0.6724807620048523, -0.7186823487281799, 0.055810414254665375, 1.4116555452346802, -0.15169034898281097, 0.9744747281074524, -0.40232276916503906, 1.1373076438903809, -1.6863377094268799, -0.33626702427864075, 0.597430408000946, 0.16420774161815643, -1.3807975053787231, 0.7801021933555603, -0.2737329602241516, -0.7786167860031128, 0.7931581735610962, 0.10832918435335159, -0.13323913514614105, 0.6443074345588684, 0.37735119462013245, 0.9472364187240601, 1.3534260988235474, -0.3193199038505554, -0.2570398151874542, 0.7245341539382935, -1.7120782136917114, -1.3095072507858276, -1.1118943691253662, -1.2831590175628662, -0.5356055498123169, -0.885412335395813, -0.6414216160774231, 0.15048028528690338, 0.32588839530944824, 1.4928337335586548, -0.8736806511878967, -0.13173650205135345, -0.10177318006753922, 0.8921354413032532, 0.015612971968948841, 1.488703966140747, 0.35318881273269653, 0.6767394542694092, -0.784730076789856, 0.09196467697620392, -0.577505886554718, 0.02553914673626423, 1.6985068321228027, -0.21886877715587616, -0.2619972229003906, -0.16544559597969055, 0.24693578481674194, 2.656114339828491]], "dtype": "float32"}, {"__nd__": [[-1.8091061115264893, -0.6948792934417725, 0.40647828578948975, 2.306994915008545, 1.2399117946624756, 0.6433746218681335, -1.0758765935897827, 0.670339047908783], [0.5265371799468994, 0.10353320837020874, 1.363553762435913, -0.7377407550811768, -0.8680461645126343, 0.6838671565055847, 0.24458619952201843, 0.03330332040786743], [1.230613112449646, 0.20199836790561676, 0.8234579563140869, -0.5219626426696777, -1.444159984588623, -1.1941559314727783, -0.36585375666618347, 0.5806864500045776], [0.6981214284896851, -0.42825740575790405, 0.5588656067848206, 0.6504753828048706, 0.28285545110702515, -0.3986494839191437, -0.9714895486831665, 1.1131412982940674], [-0.9713371396064758, 0.18621888756752014, 0.08882607519626617, 1.2427390813827515, 0.8146870136260986, -1.7949681282043457, -0.5750757455825806, -0.6519562602043152], [-0.49040767550468445, 0.8494340777397156, 0.8626537322998047, -0.39634576439857483, -2.3712151050567627, 0.0078062573447823524, 0.1650521159172058, -0.8690041303634644], [0.21762801706790924, 1.2254869937896729, -1.5064469575881958, 1.407138705253601, -0.9004756808280945, -1.0049084424972534, 1.0989809036254883, -0.48094287514686584], [-0.07063209265470505, 1.4679220914840698, 0.3153096139431, -0.03519536554813385, -2.299118995666504, -0.7261022925376892, 0.5362431406974792, 1.5415323972702026], [0.8141851425170898, -0.002299713436514139, -1.580702781677246, -1.3913475275039673, 0.7799155116081238, -0.6048325896263123, 0.635786771774292, 1.5707004070281982], [-0.7660084962844849, -0.06843272596597672, -0.26128995418548584, -0.5160800814628601, -0.9941498637199402, 1.1501884460449219, 0.11991576105356216, 0.08442338556051254], [0.6201698184013367, 0.3339812159538269, -0.16655954718589783, 0.4732794165611267, 2.7505154609680176, -0.7269707322120667, 0.7570191621780396, -2.1141083240509033], [0.8501027226448059, 0.7937332987785339, -0.40587717294692993, -0.9486197233200073, 0.22580625116825104, 0.5969972014427185, -0.9833894968032837, 1.3756232261657715], [0.8460224270820618, 0.23021049797534943, 0.23349405825138092, -0.8766602873802185, -2.205461263656616, -0.5349118709564209, 0.9949935078620911, 0.43224191665649414], [-0.9902381300926208, -1.5166277885437012, -1.3490617275238037, -0.5597916841506958, 0.220199316740036, 1.4474846124649048, -0.1340591162443161, 1.3519418239593506], [-1.5052523612976074, -0.34249553084373474, 1.1129578351974487, 0.97060227394104, -1.4638686180114746, -0.8204700946807861, 0.39158686995506287, -0.8300517201423645], [-1.0727167129516602, 0.31635725498199463, -0.6632668972015381, 1.2452059984207153, 0.5482382774353027, 1.0380114316940308, 0.3941023647785187, -0.014910115860402584], [-0.9234539270401001, 0.026851270347833633, -1.1865026950836182, -1.9761375188827515, -1.000963568687439, -2.385420560836792, -0.5546857118606567, 1.8953016996383667], [0.09220247715711594, 0.06268385052680969, -1.203834056854248, 0.026450203731656075, -0.8981626033782959, 0.1540631800889969, -0.5891799926757812, 0.6098425984382629], [-0.6265603303909302, -0.945278525352478, -1.7015907764434814, 0.2987750470638275, 0.8226641416549683, -1.3127621412277222, 1.2097163200378418, -1.4547765254974365], [1.613891363143921, -0.2060353010892868, 1.252447247505188, -0.722349226474762, 0.19805288314819336, 0.24197322130203247, 0.2698204517364502, 0.08802270144224167], [-1.2202790975570679, 0.0813620314002037, -1.3973190784454346, 0.7294901013374329, -0.05183856561779976, 0.3635839521884918, 0.3979825973510742, 0.5356864929199219], [-0.7382996082305908, -0.5238083600997925, 0.7972266674041748, 1.994542121887207, -1.3111283779144287, -0.3720335364341736, 1.1259576082229614, 0.557580292224884], [1.0916558504104614, 3.6258904933929443, 0.1790620982646942, -0.380695641040802, 1.230453610420227, -1.853821873664856, -1.568334698677063, 0.44091978669166565], [1.0768026113510132, 1.0571738481521606, 1.8338030576705933, 0.13067086040973663, -0.9304790496826172, -0.8367904424667358, -0.6071298122406006, 0.03219439834356308], [0.7837528586387634, 0.5720712542533875, 0.6823760867118835, 0.05781150236725807, -0.691135585308075, -1.2196996212005615, -0.4517594873905182, -1.4284347295761108], [-0.2691287696361542, -0.3201235830783844, -0.0867650955915451, 0.8768330812454224, 0.7897236347198486, -0.4846607744693756, -1.0083101987838745, 0.33625689148902893], [-1.0124672651290894, -2.129507303237915, -0.2674098312854767, 0.5365221500396729, -0.1004168912768364, -0.5080202221870422, 2.242072343826294, 1.7658727169036865], [0.007554605603218079, 0.001792383729480207, -0.06381218880414963, 2.1587421894073486, 0.4174545705318451, -0.9452450275421143, 0.43003252148628235, -0.6010690331459045], [-0.6604428291320801, 0.20388029515743256, 1.889857292175293, -1.1869118213653564, -0.330552339553833, 1.515509009361267, -0.6101104021072388, -0.621789813041687], [-0.29073405265808105, 0.09378428012132645, 0.1320570856332779, -1.0668598413467407, -0.8888223767280579, 0.4356362819671631, -0.9776818752288818, -0.1621691882610321], [0.13646546006202698, -0.932871401309967, 0.040549807250499725, 0.9348934888839722, 0.9600313305854797, -0.632377564907074, -1.6575956344604492, 0.48416852951049805], [-0.08493825048208237, -0.9944915175437927, 0.47631317377090454, -0.29301419854164124, -0.7976896166801453, 0.25605249404907227, 0.48264431953430176, 1.1664296388626099], [0.7041112184524536, 0.15068615972995758, 1.1222811937332153, 0.004767184611409903, -0.2955387234687805, 0.24552440643310547, 1.2238534688949585, -0.3533598780632019], [-1.7458847761154175, -0.016392990946769714, 0.2704458236694336, 0.294861763715744, -0.1528339982032776, -0.3781193792819977, -0.27193981409072876, -0.1294824182987213], [-1.5966582298278809, 0.63450026512146, 0.5198389291763306, -1.5947710275650024, -0.9410781264305115, -0.43730637431144714, 1.3397712707519531, 1.0944205522537231], [-0.7785578966140747, -0.917733371257782, -1.4984486103057861, 2.449680805206299, 0.06807775795459747, 0.1965002417564392, 0.47590523958206177, 0.8654459714889526], [0.778090238571167, -1.5549687147140503, 0.18194961547851562, -1.5579280853271484, 0.43272337317466736, -0.9624432325363159, -1.205918550491333, 1.3128584623336792], [2.0873100757598877, 1.1854004859924316, 0.2802239656448364, -0.026028525084257126, -0.3518245220184326, 1.46570885181427, -0.25378355383872986, 0.780828058719635], [-1.0246251821517944, 0.5487337112426758, 0.625469446182251, -0.43388745188713074, 1.2168817520141602, -0.34444546699523926, 0.18417119979858398, 2.3870303630828857], [0.31311067938804626, 2.3281068801879883, 0.4827185571193695, 0.21617868542671204, 0.15249590575695038, 2.7465457916259766, -0.5255656242370605, 0.6904684901237488], [0.6460906267166138, -0.5815111994743347, -0.8300760984420776, -0.637560248374939, 0.5492370128631592, 0.7992208003997803, -0.9260095953941345, -0.5712098479270935], [0.8507535457611084, -0.3568441867828369, -1.6270620822906494, -0.7022570371627808, -1.8227460384368896, 0.6516301035881042, -1.2575793266296387, -0.3148297667503357], [-1.3527003526687622, 0.04890337213873863, 0.4995998740196228, -0.34196075797080994, -0.8102198243141174, -1.7512588500976562, 0.68059903383255, 1.2195340394973755], [0.2853304147720337, 1.3184449672698975, -1.4523611068725586, 1.3489435911178589, 0.8996146321296692, -1.4617093801498413, 0.6095547676086426, -1.2820556163787842], [-1.1297056674957275, 0.5395038723945618, 0.9915993213653564, -0.36846911907196045, -0.380628764629364, -0.585109293460846, -0.9490693211555481, -0.5637436509132385], [1.856824278831482, 0.2825678288936615, 0.902435839176178, 0.5813419222831726, -0.5918523669242859, -1.233176589012146, 1.031921148300171, 0.12374989688396454], [-0.015030942857265472, 0.6212735772132874, -0.9903043508529663, 0.8711163401603699, 1.049239993095398, 0.3014092743396759, -1.8570926189422607, -0.6005163192749023], [0.5481534600257874, -0.9638146758079529, 0.935882031917572, -1.4160784482955933, -0.9670756459236145, -2.5500385761260986, 0.6903890371322632, 0.6135721206665039], [-0.2809017300605774, -0.15580037236213684, -2.3889710903167725, 1.9766143560409546, 1.743695616722107, -0.2076837420463562, 0.49212539196014404, -0.03261943906545639], [-1.1292777061462402, 0.9260947704315186, 0.019170744344592094, 0.5442391037940979, 1.2241898775100708, 0.971310555934906, -1.4608224630355835, 0.9985963702201843], [1.0197243690490723, 0.97822505235672, -0.13552848994731903, -0.8880148530006409, -0.1879362314939499, -0.3303149938583374, 0.2915581166744232, -0.7041124105453491], [1.3768147230148315, -0.09520170837640762, 1.6546130180358887, -0.4763294458389282, 1.9213283061981201, 1.5479532480239868, 0.26643598079681396, -1.7527458667755127], [0.5252392292022705, 1.8537453413009644, -0.7932199239730835, 0.6845026016235352, 1.5091819763183594, -0.6001802086830139, 0.22606366872787476, 0.16111481189727783], [0.7867168188095093, 0.00925414077937603, -0.6009082794189453, 2.198272943496704, -0.4581240117549896, 0.49076342582702637, 0.6658249497413635, 2.0180275440216064], [-0.9906988739967346, 1.339707612991333, 0.5917322039604187, -0.4413715898990631, 1.9057594537734985, -0.22147159278392792, -0.5094695687294006, -1.0221216678619385], [-0.7624210715293884, 0.7886781692504883, -0.5183382034301758, -0.9625428915023804, -1.1787786483764648, 0.11303749680519104, -1.8713018894195557, 0.4962429702281952], [-0.7267802953720093, 0.2502351999282837, 1.2785624265670776, -0.9882543683052063, -0.31983017921447754, -0.453049898147583, 0.8186636567115784, -1.3201707601547241], [1.3820183277130127, 0.042761776596307755, 1.9997721910476685, -0.4743110239505768, -0.2747131586074829, 2.1265878677368164, 0.5319873690605164, -2.6790850162506104], [1.903682827949524, -0.18944379687309265, -0.9236096143722534, -0.2633008062839508, 0.5931804776191711, 0.3635464310646057, 0.7460119128227234, -0.986807644367218], [-1.2407124042510986, 0.6264866590499878, -0.35467734932899475, 1.6639912128448486, -1.589328646659851, 1.4413594007492065, 0.34038472175598145, -0.26212450861930847], [1.19624924659729, 1.6989413499832153, -1.0637235641479492, -0.8588131070137024, 1.8113328218460083, -1.348271131515503, -1.201859951019287, -0.3253440260887146], [1.4003123044967651, -0.4476854205131531, 0.8316044807434082, -0.5281243920326233, -0.010035393759608269, -0.5041645765304565, 2.6793153285980225, -0.5568931102752686], [-0.5626509785652161, 1.4269123077392578, 2.4523122310638428, 1.5118482112884521, -0.7196127772331238, -0.3601847290992737, -0.33094629645347595, -1.2208713293075562], [0.20735901594161987, 0.0991843119263649, -0.2308119386434555, -0.551986038684845, -0.12310008704662323, 0.46645864844322205, 0.013140229508280754, -0.7330962419509888], [0.5001953840255737, 0.3273756504058838, 0.07590825110673904, -0.8681603670120239, 0.8565477132797241, 1.2011903524398804, 0.627508282661438, 0.8142824769020081], [0.8559651970863342, 0.4698215425014496, -0.34286293387413025, -0.6027312278747559, 0.42337846755981445, -1.328116774559021, -0.9450412392616272, -0.8134448528289795], [0.2293812334537506, 0.6901359558105469, 0.8732867240905762, 0.6684488654136658, 0.7066860198974609, -1.1804590225219727, -1.1331979036331177, 0.20016886293888092], [1.8046200275421143, 0.16713528335094452, -0.9834719896316528, -0.8376818299293518, 0.32030338048934937, 0.3032011389732361, -1.1401089429855347, 0.36993083357810974], [0.5697457790374756, -2.046727180480957, 2.077327251434326, 0.7906930446624756, 0.9873537421226501, -0.7491164803504944, 0.17371225357055664, -1.0709521770477295], [-0.013158421963453293, 1.794898271560669, 0.9154353141784668, -0.10714758932590485, 0.9096369743347168, 0.701914370059967, 0.2898271381855011, -0.09919299930334091], [-2.089491844177246, -1.4215830564498901, 1.6232264041900635, -1.7637540102005005, 1.830321192741394, 0.025194501504302025, -0.5630939602851868, 1.3170853853225708], [0.15073442459106445, -0.9257817268371582, 0.8688603639602661, 0.8835732936859131, -0.6770148873329163, 0.709949791431427, 0.28312090039253235, -0.9093376398086548], [0.5621681809425354, 0.41530725359916687, 1.563023567199707, 0.014212195761501789, -0.05983465164899826, 1.6291579008102417, 0.8910653591156006, 0.25082775950431824], [-1.6178690195083618, -1.6246883869171143, -2.188617706298828, -1.3077985048294067, -0.15324831008911133, -0.26414430141448975, 0.45731741189956665, 0.40883252024650574], [-1.1201521158218384, 0.7133257985115051, 0.7115640640258789, 0.12097206711769104, -1.417490005493164, -0.40259239077568054, -0.8600068688392639, 0.11186755448579788], [0.45495128631591797, -0.8459075689315796, 1.572661280632019, -1.219997763633728, -1.14506995677948, 0.22349479794502258, 0.522341787815094, -1.0699504613876343], [-0.10414556413888931, 0.2829664349555969, 3.208117961883545, 0.8262491822242737, -0.6878166794776917, -0.031139379367232323, 0.8103705644607544, 0.18151919543743134], [-0.01918429508805275, 2.6511340141296387, -0.5274650454521179, -0.500254213809967, 0.35496029257774353, 0.46047526597976685, -1.9742677211761475, 0.5166732668876648], [1.2880852222442627, -1.0865178108215332, 1.9429455995559692, 0.361730694770813, 0.6927216649055481, 0.3537057042121887, -0.08624015003442764, 0.5319026112556458], [0.06900115311145782, -0.2226216346025467, 1.3407703638076782, -0.47603994607925415, -1.1045926809310913, 1.4718390703201294, 0.02129622921347618, 0.08480208367109299], [-0.37625041604042053, 0.9631717205047607, -2.6024727821350098, 0.5769737958908081, -0.7528854012489319, 0.7078498601913452, 0.12853950262069702, -0.01346401683986187], [0.17255394160747528, -0.4861305356025696, 1.2320244312286377, -0.22318977117538452, 1.134985327720642, 0.3687249720096588, 1.4035511016845703, 0.7469407320022583], [-0.0539030097424984, 0.32195380330085754, 1.042235016822815, -0.9063793420791626, 0.5214360952377319, 0.4052804410457611, 0.26330938935279846, -1.1787216663360596], [0.3524930477142334, 0.7336450219154358, -1.38468599319458, 0.18329082429409027, -0.8133454322814941, -0.7712764143943787, 0.3193475008010864, 0.12655219435691833], [-1.3572101593017578, -1.287247657775879, 1.864642858505249, -0.5050703287124634, -0.6941457390785217, -0.5956059694290161, 0.0192774198949337, -0.4364711046218872], [0.05630064383149147, 1.7746208906173706, -0.6301182508468628, 0.8203253149986267, -0.721439003944397, 0.286981463432312, 0.08377330005168915, 0.421352356672287], [1.5909618139266968, -0.11662296950817108, -0.5838017463684082, 1.266735553741455, 0.2938997447490692, 0.20025739073753357, 0.6275755167007446, 1.7443422079086304], [-1.423750638961792, 0.5064162611961365, -2.613846778869629, 1.1568156480789185, 0.9290058612823486, 0.8164637088775635, -0.1859724521636963, 0.5449294447898865], [-1.3416521549224854, -1.0712380409240723, -0.3398725986480713, 0.5008113980293274, -0.30569344758987427, -0.5699110627174377, -0.07584945112466812, -1.292648434638977], [-0.9609613418579102, 0.6683031320571899, -0.6479414701461792, -1.170441746711731, -0.10690134018659592, 2.0821828842163086, 1.0745769739151, -1.5600870847702026], [0.02375771664083004, 1.732834815979004, -1.6796739101409912, -0.47540315985679626, 1.0831035375595093, 0.16418339312076569, 0.9096715450286865, -1.4750003814697266], [-1.5176076889038086, -1.7294394969940186, -0.36307594180107117, 1.061672329902649, 0.706457793712616, 0.9793379902839661, 0.439042866230011, -0.5203768610954285], [1.8244359493255615, -0.6153201460838318, -0.664047122001648, 2.2416486740112305, 1.2991739511489868, -1.7287958860397339, 0.4923052191734314, 1.4516510963439941], [2.4528908729553223, 0.31136325001716614, 0.18981343507766724, -0.7472980618476868, 0.7536646723747253, 1.0565528869628906, -0.50569748878479, 0.3380122780799866], [1.882775902748108, -0.1493413746356964, -1.0528355836868286, 0.046651218086481094, 0.36046698689460754, -0.4650438129901886, 1.2559162378311157, -0.6715554594993591], [0.92079758644104, -0.4987412393093109, 2.241767406463623, -1.8118213415145874, -0.7292364239692688, -0.15351974964141846, 0.04949933663010597, 0.33362579345703125], [0.20676212012767792, 1.0058248043060303, 0.6671748757362366, 0.7025176286697388, -1.6367242336273193, -0.44642430543899536, 0.5624484419822693, -1.6275578737258911], [0.6430065631866455, 1.7421271800994873, 0.5064421892166138, 0.3927028477191925, -0.4027906060218811, 0.10142185539007187, -2.1473381519317627, -0.5211856365203857], [1.6567060947418213, 1.8046218156814575, -0.5868932008743286, -0.533297598361969, 0.9657196402549744, -0.4372674524784088, -2.271164655685425, -0.5368590950965881], [0.10919061303138733, 1.0188517570495605, -0.8728699088096619, 1.078896164894104, -0.27709582448005676, 0.8020364046096802, -0.23194444179534912, -0.8619430661201477], [0.20595695078372955, -0.46069562435150146, 1.3193503618240356, 0.5362458825111389, -1.7088760137557983, 0.16692687571048737, -1.4872400760650635, -0.273436039686203], [-0.029613541439175606, -0.856357753276825, 0.04572592303156853, 0.8705607652664185, -0.4710141122341156, 0.33376961946487427, 1.788841962814331, -1.9061115980148315], [-0.3745274543762207, 0.10113703459501266, 0.36539924144744873, -0.13930894434452057, 1.4309803247451782, -0.18514108657836914, -0.9150019884109497, -0.4672510027885437], [0.07383160293102264, -1.266390323638916, 0.3581775724887848, -1.236291527748108, 0.2258693128824234, 0.3668924570083618, -2.2026002407073975, -0.6296062469482422], [0.5408006906509399, 0.48395106196403503, -0.9115399122238159, 0.9833457469940186, -0.001425003632903099, -1.2921934127807617, -0.4440144896507263, -0.10477577149868011], [0.6589313745498657, -0.3987525999546051, -1.5026724338531494, 1.8151578903198242, 1.4035041332244873, 1.041152834892273, -1.9092472791671753, -1.2301946878433228], [-0.17099282145500183, 0.2269759476184845, 0.36439916491508484, -0.10454782098531723, -1.8195480108261108, -0.2926601767539978, -0.8473390936851501, 1.6929974555969238], [0.36765047907829285, 0.7123465538024902, 1.946215271949768, -0.5196326375007629, 1.018608808517456, -0.17392483353614807, -1.1775736808776855, -1.1861146688461304], [-1.159777283668518, 0.08554454892873764, 1.1186383962631226, 1.2704168558120728, 0.37948375940322876, -0.23221850395202637, -1.8868403434753418, -0.017127789556980133], [-1.5212959051132202, -1.2460538148880005, -0.011341334320604801, 0.4620613753795624, 1.070072889328003, -0.08126316964626312, -2.2176952362060547, -0.08361570537090302], [-0.12371766567230225, 1.5318559408187866, -1.327572226524353, 1.0601140260696411, -0.025323381647467613, 2.6671502590179443, 0.3568519055843353, 0.7698872089385986], [-0.09178713709115982, 1.259093999862671, 0.21046920120716095, 0.18371941149234772, -1.747003197669983, -0.22444362938404083, -0.8933013677597046, 1.698796033859253], [0.5592896938323975, 1.4221038818359375, 0.0031861586030572653, 0.09271863102912903, 0.3642396628856659, 0.793137788772583, 0.2472517341375351, -0.957430899143219], [1.7460561990737915, -0.3778022229671478, -1.0281482934951782, -0.8177157044410706, -0.6827114820480347, 0.43357419967651367, 1.0982743501663208, 0.08823879808187485], [-1.0357729196548462, 0.6960313320159912, 2.146876335144043, -0.695378839969635, 0.2908768057823181, -0.5039680600166321, 1.2060775756835938, 0.10426846146583557], [0.7544095516204834, 0.23322470486164093, 1.9753867387771606, -0.7782762050628662, -0.4755915403366089, -1.4185689687728882, 0.24171501398086548, 1.6410292387008667], [-0.6410962343215942, -0.21031257510185242, 0.2222641110420227, 0.1851232498884201, 0.19957610964775085, 0.4114631116390228, 1.1666456460952759, -0.4797534942626953], [0.35631194710731506, 0.6863646507263184, 0.4479530155658722, 0.23786601424217224, -0.3471734821796417, -0.891596257686615, 0.2304663211107254, 0.01767769642174244], [-0.6159993410110474, -1.2789239883422852, 0.20413844287395477, 0.3228779137134552, 0.12629076838493347, 0.600950300693512, -0.5262951850891113, 0.6482349634170532], [0.9704638719558716, -0.639819860458374, 0.6990261077880859, -0.23770153522491455, 0.769363284111023, 0.15152710676193237, -0.7698807120323181, 1.9080396890640259], [1.352946162223816, 1.4826542139053345, -0.9099506139755249, -1.185228943824768, -0.6585142612457275, 1.9646552801132202, -1.3300169706344604, -0.75830078125], [-0.8233723640441895, -0.903008222579956, 0.26867571473121643, 0.46992063522338867, 0.8683251142501831, 1.0931607484817505, 0.051345232874155045, 0.4657188951969147], [-0.9945974349975586, 0.5782656669616699, -0.5076309442520142, -0.08510556817054749, 0.028780950233340263, 0.6370168924331665, 0.023040270432829857, 0.5497539043426514], [-1.6499696969985962, -0.7583009600639343, 0.9294679760932922, -0.5475252270698547, -0.2254086434841156, -1.923744797706604, -1.0958268642425537, 0.23816393315792084], [-1.2125879526138306, -0.10938292741775513, -0.3572799861431122, 1.798755168914795, 0.6430290341377258, 2.3726840019226074, -1.9269253015518188, -1.145229697227478], [-0.020932063460350037, -1.6808900833129883, -0.9795432090759277, -0.6024267673492432, 0.16014926135540009, 0.8719850778579712, -1.3644593954086304, -1.72291898727417], [-0.27507805824279785, -0.9141498804092407, -0.00811493769288063, 1.7039798498153687, 1.9304198026657104, -0.885271430015564, 0.4194791316986084, -0.7169241309165955], [-2.14280104637146, -0.980097770690918, -0.19900241494178772, 1.222503662109375, -0.514792263507843, 0.3138439655303955, -1.248562216758728, -0.9812290668487549], [-0.4037553071975708, 1.6855647563934326, -0.8737847208976746, 1.2633754014968872, 1.1683956384658813, -0.45536744594573975, -0.3412526547908783, 1.6917691230773926], [-0.8947595953941345, 0.8325216174125671, -0.291219562292099, 0.9913647770881653, 1.5872246026992798, -2.166520833969116, 1.1985571384429932, -0.08096082508563995], [0.7784759998321533, -0.9674908518791199, 0.3832622170448303, -0.06747215241193771, -0.059046193957328796, 1.0043003559112549, -0.23703911900520325, -0.4904862642288208], [0.1885511577129364, 0.14968618750572205, 0.7125041484832764, -0.38443654775619507, 0.6190862059593201, -1.1021960973739624, 0.09830920398235321, -0.15872105956077576], [0.6123224496841431, 0.6194924116134644, 0.4028644859790802, 0.13511213660240173, 0.88169264793396, 0.4489772319793701, 0.8469637632369995, -1.2656733989715576], [0.015231398865580559, 1.009141206741333, -0.5642873048782349, 0.0041691311635077, 0.08698797971010208, 0.6760250329971313, 0.024820195510983467, -1.1128346920013428], [-0.9545204043388367, -1.0219229459762573, 0.6947751045227051, -0.31563088297843933, -0.9025273323059082, -1.3479083776474, -1.1311546564102173, -1.1835178136825562], [0.263731449842453, -0.19636587798595428, 0.5832900404930115, -0.07265068590641022, 0.2765989303588867, 0.3729473650455475, -0.8500342965126038, -0.56980299949646], [0.7025467157363892, -0.3030160069465637, 0.5526208281517029, 0.3490649461746216, -1.8812857866287231, 0.9938083291053772, 0.3670872747898102, 0.15148648619651794], [0.44955185055732727, -1.1160640716552734, -1.5062350034713745, -2.358440399169922, 1.655350685119629, 0.35192131996154785, -0.1779651939868927, -0.6315242648124695], [-0.0009444375755265355, -1.3386884927749634, 0.8560194969177246, -0.9052817821502686, -0.1415284276008606, 1.1022047996520996, -0.20901556313037872, 1.043506145477295], [0.43459656834602356, -0.9441616535186768, -0.33355075120925903, -0.6372923851013184, 1.9127188920974731, 1.1424120664596558, 0.08439330756664276, -0.3164346516132355], [1.0898339748382568, 0.6800527572631836, 1.0465140342712402, -0.4239630103111267, 1.351771593093872, 0.8490312695503235, -0.9009369015693665, 0.38200902938842773], [0.06050239875912666, 1.6096742153167725, -0.7196207046508789, 0.7112134099006653, -0.2177162915468216, -2.2535781860351562, -0.3261989653110504, -0.06532304733991623], [-0.1084180474281311, -0.5960172414779663, 1.5189942121505737, -0.06988159567117691, -0.3229502737522125, 0.6044110059738159, -0.500759482383728, 0.8741104006767273], [-1.288952350616455, -0.36207765340805054, 1.072638988494873, 0.5819505453109741, 0.284665048122406, 0.5200006365776062, 0.27197930216789246, 1.134864330291748], [-0.157740980386734, 0.5424132347106934, 0.3712180554866791, -0.6468223333358765, 0.5692635774612427, -0.14852958917617798, 0.3180184066295624, 1.789618730545044], [-0.1375243067741394, 0.4413290321826935, -1.0851837396621704, -0.14981962740421295, -0.5499888062477112, 1.956817865371704, -0.4642484486103058, -0.4551914930343628], [-1.0364748239517212, -0.6813780069351196, 0.6112709641456604, -0.33949652314186096, 1.8598380088806152, -1.079519271850586, -0.8324075937271118, 0.6768168807029724], [-0.4398081600666046, 0.04763086885213852, -0.6405632495880127, -1.4585473537445068, 1.3403867483139038, -1.5185840129852295, -0.4743490219116211, 0.9377159476280212], [-0.8820680379867554, 0.6180513501167297, -0.11894094944000244, 0.4308560788631439, -0.17504648864269257, -0.33703118562698364, 0.015809355303645134, 2.8482766151428223], [-0.20674721896648407, -0.5879995822906494, 1.0803472995758057, -0.17724503576755524, 0.14612998068332672, 0.7179349064826965, -0.12034408003091812, 0.5278449654579163], [-0.6195865869522095, -0.6216335296630859, -1.3895584344863892, -2.4171223640441895, 0.530381441116333, -1.6517475843429565, -0.7311391234397888, 0.5304789543151855], [-1.5011858940124512, 0.2613300383090973, -0.2895341217517853, 2.124282121658325, 0.5028020739555359, 0.16594184935092926, 0.35699981451034546, -0.9940590858459473], [1.5871611833572388, 1.2505900859832764, 0.779338002204895, 0.601679801940918, 0.5666240453720093, 0.26615506410598755, -2.63264536857605, -2.638550043106079], [-0.5214988589286804, 0.9966432452201843, -0.8631345629692078, -1.8941599130630493, 1.3884981870651245, 0.7335399389266968, -0.8112612962722778, 0.44031521677970886], [-0.29832449555397034, 0.2677818238735199, 0.3644002079963684, -0.45274603366851807, 0.7296082377433777, 0.33302977681159973, -0.5629299879074097, -0.7821050882339478], [0.4665013551712036, -1.1143425703048706, 0.12894998490810394, -1.452301263809204, 0.9373947978019714, -0.7775041460990906, 0.05415600538253784, -0.5648153424263], [-0.42788735032081604, 0.4130082428455353, 0.08424214273691177, -1.1228059530258179, -0.9847894310951233, -0.6806371212005615, 0.6268003582954407, 0.5905687808990479], [0.695343554019928, 0.8792303800582886, -0.6402199864387512, -0.24401246011257172, -0.8037450909614563, 0.1196865439414978, -0.7485033869743347, -0.04931812733411789], [0.01862148754298687, -0.2859671413898468, -0.9332939386367798, -0.5359026193618774, -0.9788044691085815, -1.398152232170105, -0.1281561255455017, -0.22881990671157837], [-1.987694501876831, -0.029154937714338303, 2.4579720497131348, 0.35902997851371765, 0.18635903298854828, -0.44746893644332886, 0.7914049029350281, -0.3114897310733795], [0.20004478096961975, 0.4228821396827698, 0.7883379459381104, 0.4162251055240631, -0.25358277559280396, -0.2576914429664612, -1.4506471157073975, -0.8583999276161194], [-0.14219577610492706, -0.7252718210220337, -0.5623708963394165, -2.811981201171875, -0.1806917041540146, -0.03861372172832489, -0.736581563949585, 1.3194583654403687], [1.3296154737472534, 0.1770791858434677, -0.6962448954582214, -1.009661078453064, -2.008700370788574, 2.9125354290008545, 0.03405033051967621, 0.709854245185852], [-1.928911566734314, 1.4482390880584717, 1.2038129568099976, -0.08726362884044647, 0.04041944444179535, 0.9501346945762634, 1.4219011068344116, -1.6565005779266357], [2.761087417602539, -0.1761157363653183, 1.7384315729141235, 0.410762757062912, -0.4767882525920868, -0.16996999084949493, 0.9170942902565002, -2.2112178802490234], [-0.20732595026493073, 1.2444016933441162, -0.6968103647232056, -0.347159743309021, -0.027943968772888184, -1.762538194656372, -1.263563871383667, 0.08898182958364487], [1.4920220375061035, -1.1111193895339966, -0.9704895615577698, -1.0010277032852173, 0.6066544055938721, 0.33683696389198303, -0.13109400868415833, -0.07279585301876068], [0.0026908947620540857, 1.9378209114074707, 1.2095799446105957, -0.9147128462791443, -0.009451832622289658, -0.7971434593200684, -1.1203958988189697, -0.47328126430511475], [-0.6815357804298401, -2.255026340484619, -0.21007569134235382, 0.46820393204689026, 0.7887874245643616, 0.5123413801193237, -0.6118924021720886, -0.7658868432044983], [2.140507936477661, 0.3936231732368469, 0.033654261380434036, -0.8921194672584534, 2.3434762954711914, -0.6108322143554688, -0.6057718396186829, 0.8054404854774475], [-2.5926434993743896, 1.8230226039886475, -0.18035824596881866, 0.19590488076210022, 0.95087069272995, 0.5919889807701111, -1.0328086614608765, 0.17750367522239685], [-0.8381165266036987, -0.021919341757893562, 1.281882643699646, -0.20413516461849213, 0.12996558845043182, 2.194648265838623, 1.912145733833313, 2.5259528160095215], [-0.08141310513019562, 0.3887469172477722, 0.7530346512794495, 1.8498343229293823, -1.1102149486541748, -0.013123055920004845, 1.077938437461853, -0.9657742977142334], [1.1570403575897217, -0.15818250179290771, 0.17020803689956665, 0.36949172616004944, 1.94888174533844, -0.07009831815958023, -1.0566515922546387, -2.3593263626098633], [-0.28415271639823914, -0.8954117298126221, 0.14882177114486694, 0.7083882093429565, -0.31593915820121765, -0.14150302112102509, -0.7554839849472046, -0.9355820417404175], [1.0103179216384888, -0.45941609144210815, 1.1217023134231567, -0.24076147377490997, -0.8284017443656921, -0.2767406105995178, 0.4407317340373993, 0.7891645431518555], [-1.8193880319595337, -1.1984076499938965, 0.6576281189918518, 0.6559277772903442, -0.1267486959695816, 0.8934386372566223, -0.4964525103569031, -0.8349632024765015], [-0.8506271243095398, 1.2087712287902832, 0.018267931416630745, -2.6213347911834717, 0.15210074186325073, -1.137427568435669, 1.0136959552764893, 0.6603850722312927], [0.08491736650466919, -1.203634262084961, -0.6076046824455261, 0.8945364356040955, -0.16860714554786682, -0.7388626933097839, -2.3776516914367676, -1.1133825778961182], [1.1165835857391357, 1.2014015913009644, 1.1022326946258545, 0.36664021015167236, -0.08341681957244873, -1.684697151184082, 0.008012608624994755, 0.39483317732810974], [2.3964338302612305, -0.8034241199493408, 0.23162798583507538, -1.9004443883895874, 0.5137620568275452, 0.13966476917266846, 3.021608829498291, -0.3153054714202881], [-1.411253809928894, -0.11852093786001205, -0.5951050519943237, -0.36844614148139954, 0.10460153222084045, 2.756256103515625, 0.39173731207847595, -0.15937311947345734], [0.641686201095581, 0.5025123953819275, -0.47385987639427185, 0.9347146153450012, 1.0700691938400269, 1.6994726657867432, -0.3154020607471466, -1.0152723789215088], [0.4946848452091217, -1.2454378604888916, -0.00904060248285532, -0.5682109594345093, -1.1716899871826172, -1.9054652452468872, -0.07998691499233246, -0.1764497458934784], [-1.1676008701324463, 0.6415720582008362, 0.9105142951011658, 0.5276011824607849, 0.7679852247238159, 1.168854832649231, 1.6014125347137451, -0.01795954816043377], [0.9028361439704895, -0.4215417504310608, 0.39150527119636536, 0.10598764568567276, -1.8428822755813599, 0.6651989817619324, -1.3243995904922485, -0.3495842516422272], [-0.378814697265625, -1.625175952911377, -1.161210536956787, -1.0512051582336426, 0.0201487448066473, -1.2515573501586914, -0.38328665494918823, -0.5303913354873657], [0.8157851099967957, -0.16374172270298004, -0.24631084501743317, -1.6687769889831543, -0.8413463830947876, 0.6076480746269226, -0.21419726312160492, 0.31569862365722656], [1.5904263257980347, -1.105205774307251, -0.589925229549408, -1.5415184497833252, 0.21485978364944458, -0.34807777404785156, -0.00720023550093174, 0.335222065448761], [-1.4140583276748657, 0.6386466026306152, 0.500047504901886, 0.6465128660202026, 1.3079257011413574, -0.07724599540233612, -0.18257948756217957, -0.3044450879096985], [0.3251415491104126, 1.4639780521392822, 0.5698438882827759, -2.8460607528686523, -0.19586166739463806, -1.0503244400024414, 0.2908268868923187, 1.3751343488693237], [-2.511143684387207, -0.09304551035165787, 0.7612640857696533, -0.20797553658485413, 0.492837131023407, 2.323976516723633, -0.031516190618276596, -0.652353823184967], [2.9740066528320312, -0.4239003360271454, 0.6393640637397766, -0.983996570110321, 0.4218790829181671, -1.5929198265075684, 0.4104182720184326, 1.4042521715164185], [-0.6187072992324829, 0.2109520137310028, -1.544141411781311, 1.2092833518981934, -0.929631233215332, -1.5270155668258667, -0.6307512521743774, -0.0090468879789114], [-0.9157630801200867, 1.036275863647461, 0.4488826096057892, -2.169754981994629, 0.45801976323127747, -1.793839693069458, -2.8634426593780518, -0.16157963871955872], [0.14124125242233276, -0.24937833845615387, -0.19041267037391663, -0.06956256926059723, 0.21427205204963684, -0.291299968957901, 0.10388512164354324, 1.9961020946502686], [0.7753354907035828, -1.2866082191467285, -0.4171789586544037, 1.4193485975265503, -0.8544335961341858, -0.29537540674209595, -0.19459611177444458, 0.9641969799995422], [-0.021113181486725807, -1.4143136739730835, 0.109763503074646, -0.9278010129928589, -1.1849040985107422, 0.7310163378715515, 0.5321818590164185, -0.46428272128105164], [0.9737992286682129, 0.1339077204465866, -0.47530248761177063, -0.4336058497428894, -0.8635377287864685, 0.9331193566322327, -1.3460273742675781, 0.9843671917915344], [0.011510130949318409, 0.3890647888183594, -0.7884865999221802, 0.30133381485939026, 0.041109368205070496, -0.4586348831653595, -1.173987627029419, -0.4171799421310425], [0.05852973088622093, -1.1796212196350098, -0.3351207673549652, -1.2807050943374634, -0.11030028015375137, -0.3745615780353546, 1.0657051801681519, -0.753398060798645], [0.8768109679222107, 0.6977279782295227, -0.8076303601264954, 0.5483157634735107, 0.6926255226135254, 1.8456530570983887, -0.8379318714141846, -1.022286057472229], [0.011809133924543858, 0.7727862596511841, -1.4608955383300781, -1.1490952968597412, 2.0462934970855713, 0.19765642285346985, -0.5448010563850403, 0.08668969571590424], [0.7635260820388794, -1.320897102355957, -0.2564253509044647, -1.6624488830566406, -0.6373555064201355, 0.3455386459827423, 0.5446860790252686, -2.1593270301818848], [-1.8336081504821777, -1.1600878238677979, -1.7438931465148926, 0.49778124690055847, -0.21995633840560913, 1.0564136505126953, -1.5408744812011719, 0.36626964807510376], [1.1748498678207397, 0.3157300353050232, 0.5050640106201172, 1.0902738571166992, -0.3707434833049774, 0.3023267388343811, -1.224068284034729, -2.14302921295166], [1.0721122026443481, -1.3695875406265259, -0.5630028247833252, 0.36349815130233765, -1.118181586265564, -0.2713063955307007, 0.5162196159362793, -0.2393958866596222], [-0.34522750973701477, 1.5013474225997925, -0.44214218854904175, 0.4047297239303589, 0.020399458706378937, -1.4547228813171387, -1.4885793924331665, -0.34763866662979126], [-1.2204753160476685, 1.163216471672058, 1.0723387002944946, -1.09635329246521, -2.480863571166992, 0.8252143859863281, 0.036047399044036865, 0.9013445377349854], [0.7296936511993408, -0.8221355080604553, 0.3627203106880188, -1.296182632446289, 2.0928685665130615, -2.1290292739868164, -0.16295106709003448, -0.009031313471496105], [0.223809614777565, 0.39574652910232544, 0.5548797845840454, -0.36936596035957336, -1.169821858406067, -0.04369831457734108, -0.4895671308040619, 1.634895920753479], [0.18578647077083588, 0.23500403761863708, -1.163192629814148, 0.794478714466095, -0.37389546632766724, -0.120643749833107, 0.7320036888122559, 0.7793755531311035], [0.015312984585762024, 0.289379358291626, -0.6284270286560059, 0.5483136177062988, -1.1666603088378906, -0.5293357372283936, -0.03986053541302681, 0.7495414614677429], [1.4077988862991333, -0.6358193755149841, -0.33782297372817993, -0.2629289925098419, 0.5921463370323181, -0.0664362758398056, 0.5895664691925049, 0.9633809924125671], [0.2969822287559509, 0.9328383207321167, -0.10775784403085709, -0.1551610231399536, 0.7996888756752014, 0.5390059351921082, 0.8719013929367065, -1.6063588857650757], [0.22133195400238037, 1.4316881895065308, -1.061457872390747, 0.749248743057251, -0.7414993643760681, -1.4548283815383911, 0.9955520629882812, 0.6417886018753052], [-0.25413650274276733, 0.0006210154970176518, 0.10398479551076889, -0.23388995230197906, 0.39027804136276245, 1.0804815292358398, 0.45671430230140686, -0.6527895331382751], [-2.831446647644043, 0.30899691581726074, -0.4859631061553955, -0.2366228997707367, 0.5472438335418701, -0.49568602442741394, -0.18778753280639648, 0.04127994924783707], [-0.8327833414077759, 1.1807405948638916, 0.325443297624588, 0.2603563368320465, 0.4914170801639557, -0.13101592659950256, -1.235203742980957, 1.4542570114135742], [0.43914273381233215, 0.6525079011917114, 1.4532843828201294, -0.38851451873779297, 0.04644493758678436, -0.42527127265930176, 0.5513038039207458, 0.7237297296524048], [0.6294272541999817, -0.22182314097881317, -1.3094209432601929, 0.3593394458293915, 0.7119985222816467, 0.9373540282249451, 0.7625026106834412, 0.6130746006965637], [0.1648845374584198, -0.12286747992038727, 0.20053990185260773, 1.1892796754837036, -1.0331275463104248, 0.7364597320556641, 0.217549666762352, 0.04063405841588974], [-1.0170869827270508, 0.690739631652832, 0.09380439668893814, 0.9148287773132324, 0.4520924687385559, 1.0617053508758545, -1.7394038438796997, -0.9537864327430725], [-1.4335393905639648, 0.776508092880249, -0.0870562419295311, 0.4960384964942932, -0.7747814059257507, 0.4390399158000946, 0.9033698439598083, -1.2211976051330566], [-1.2257905006408691, -0.5704699754714966, 3.008784770965576, -0.8353262543678284, -0.24300450086593628, -0.42030951380729675, -0.29167357087135315, -1.5508934259414673], [-0.9403021335601807, 1.4928728342056274, 0.5643149018287659, 2.4130196571350098, -0.12360353767871857, -1.867287516593933, 1.2792375087738037, -0.5472091436386108], [0.21794182062149048, -0.4841896593570709, -2.1745243072509766, 1.4856412410736084, 1.0844656229019165, -0.11241858452558517, 0.5762654542922974, 0.10492414981126785], [0.6428159475326538, -0.13786157965660095, 1.3545737266540527, 1.2046722173690796, -0.26595166325569153, 0.634956419467926, -0.6749216914176941, -2.290045738220215], [0.08513065427541733, -0.34113946557044983, 0.4384010136127472, -0.09936807304620743, -0.7836265563964844, -0.030732115730643272, -0.03196355327963829, 1.6233959197998047], [0.0500316396355629, 0.3468681573867798, -0.372214138507843, -2.544606924057007, 0.8430852293968201, 0.5866175293922424, -0.2927025258541107, 0.039361730217933655], [-1.092071533203125, 1.5052528381347656, -0.28420472145080566, -1.5470318794250488, -1.870479702949524, -0.3997061252593994, -2.1597225666046143, 0.7944006323814392], [0.9202451109886169, -1.882743239402771, -0.3725866377353668, -0.7768163681030273, -2.2250638008117676, -0.09635990113019943, -0.28875765204429626, -0.45602157711982727], [-1.3569167852401733, 0.24580436944961548, 0.010373338125646114, 0.8141520619392395, 1.4856271743774414, 0.25346800684928894, 0.0003220686048734933, -0.7207912802696228], [0.6968822479248047, 0.8627222776412964, 0.2838602364063263, 0.3473505675792694, 0.3824796676635742, -0.7185060381889343, 0.36988312005996704, 0.7970702052116394], [1.9994183778762817, -0.1899261176586151, -1.3646278381347656, -0.6035007834434509, -1.38776433467865, -0.32005155086517334, 0.6273326873779297, 0.6311349272727966], [0.18906466662883759, -1.695784091949463, -0.238302543759346, 0.13226568698883057, -0.22296050190925598, -1.9394969940185547, 1.1274166107177734, 0.3501829504966736], [-0.9948587417602539, 0.9437162280082703, -0.5490884184837341, -0.45605337619781494, -0.7020871639251709, -0.7286175489425659, -1.29897141456604, 0.7280660271644592], [1.9855983257293701, 0.4495631754398346, -0.6898014545440674, -0.25881606340408325, 1.6584498882293701, 0.5469526052474976, 0.8866094350814819, 1.9348504543304443], [-0.4835060238838196, 0.0386962965130806, -0.4675544202327728, -0.5071143507957458, -0.1990661472082138, 0.41257649660110474, 1.090190052986145, 0.05725196748971939], [-2.4394683837890625, 0.16413313150405884, 0.6425231099128723, -1.3217182159423828, 0.41594570875167847, 0.26663804054260254, 0.6878982186317444, 0.35879436135292053], [0.3917767405509949, 0.030587246641516685, -0.39852601289749146, -2.2006378173828125, 0.3364594578742981, 0.5076438188552856, 0.30279940366744995, -0.5470012426376343], [0.5018637776374817, -0.39448171854019165, 0.5894883275032043, -0.6731241345405579, -0.3070701062679291, -0.4950874149799347, -0.14763522148132324, -0.7842221856117249], [1.4585107564926147, 0.4583316147327423, -0.23108884692192078, -0.14521610736846924, -0.3418585956096649, -1.0800930261611938, -1.7386584281921387, 0.18547281622886658], [0.7224788665771484, 1.2618657350540161, -0.37445780634880066, 0.5205332636833191, 0.5739864110946655, 1.2277238368988037, 0.9569652676582336, -0.2676835060119629], [0.2873901128768921, 0.40937912464141846, -0.5161663293838501, -0.8105327486991882, -0.55150306224823, 0.759739100933075, 0.11361156404018402, -0.8041409254074097], [-0.8358434438705444, 1.487052321434021, 0.44125500321388245, -0.33778640627861023, -1.9296715259552002, 1.0802762508392334, 1.7376428842544556, 0.09428732097148895], [-2.092057704925537, -0.2702818512916565, 0.5279473662376404, 0.4662342965602875, -0.3324020802974701, -0.7100593447685242, 0.5535268187522888, 0.6569796800613403], [-0.5257842540740967, 0.9201272130012512, -1.2387179136276245, 0.5302673578262329, 0.9239591956138611, 1.4219059944152832, -0.9329825639724731, 0.43737921118736267], [2.3074779510498047, -0.1767948865890503, -0.11991357803344727, 1.7351607084274292, 0.45569586753845215, -0.0006965105421841145, 0.9016872644424438, 0.7384002804756165], [-0.0038594717625528574, -1.0089023113250732, -0.1668856143951416, -0.7123710513114929, 1.562368392944336, -0.4790680706501007, 0.69387286901474, 0.48939791321754456], [-0.6072818636894226, 2.3301279544830322, -1.014045000076294, -0.9494695067405701, -2.2305357456207275, 0.05757306143641472, 0.8324621319770813, -1.0221108198165894], [0.6975330114364624, -0.4923878610134125, -0.959575355052948, -1.3345859050750732, 2.543576240539551, 0.39641088247299194, 0.717261016368866, -1.1482852697372437], [0.07203415036201477, 0.37525805830955505, 0.5192522406578064, 0.8888890743255615, 0.2941431403160095, 0.4540256857872009, -1.1684595346450806, 1.1575955152511597], [-0.23377326130867004, -0.741804301738739, 0.6680045127868652, 0.18051794171333313, -2.566983461380005, -1.4717936515808105, 1.3211756944656372, -0.510631799697876], [-1.7604550123214722, -0.9718945026397705, -0.4058959186077118, -0.6592003107070923, -0.15773971378803253, 1.3494917154312134, -0.4982892870903015, 2.2011289596557617], [0.3301888108253479, 0.8075870275497437, 0.9095783233642578, 3.0905370712280273, -1.0649555921554565, 0.34417837858200073, 0.36767351627349854, 1.288510799407959], [1.483942985534668, 1.0992220640182495, -1.4460948705673218, -0.8443228006362915, 1.111756443977356, 0.9131349921226501, 0.9228905439376831, -1.2225074768066406], [-0.273104727268219, -1.2284824848175049, 0.8867996335029602, 0.43741199374198914, -1.0561330318450928, 2.087787628173828, 0.6868743896484375, -0.7926718592643738], [1.4551124572753906, 1.5099557638168335, -0.28477975726127625, -0.9516122341156006, 1.1197041273117065, -1.0836000442504883, -0.1669842004776001, -1.4190207719802856], [-0.9779295921325684, -1.5951521396636963, -0.9928027987480164, -0.2966829240322113, -1.2420008182525635, -0.3676171600818634, 0.06276658177375793, -1.2293351888656616], [0.8099971413612366, -0.525964617729187, 0.8787316679954529, -0.4510935842990875, -0.7626051902770996, -0.5617415308952332, 0.9622421860694885, -0.056207410991191864], [-0.9819685220718384, 0.3341510593891144, 1.3199267387390137, 0.05518108233809471, -0.31308412551879883, 0.3222358524799347, 0.2437126487493515, -0.34176626801490784], [1.8736532926559448, 0.154762864112854, 0.9170194268226624, 0.14781907200813293, -0.2534322738647461, -2.4499824047088623, 0.02818213775753975, -0.2149701714515686], [-0.5519530773162842, 0.8236075639724731, -0.02407793328166008, -1.259574055671692, 0.9643155932426453, -2.0016705989837646, -0.7835096120834351, -1.4622845649719238], [0.20068296790122986, -0.07600244879722595, -1.3078796863555908, -0.5742235779762268, 0.6360781192779541, -1.6455082893371582, -0.34838593006134033, -0.8365997076034546], [-1.8422770500183105, -0.5341362953186035, 0.47387945652008057, -0.14258618652820587, -1.3213914632797241, -2.1304452419281006, 0.09602611511945724, -0.1537790298461914], [0.23161360621452332, -0.03454361483454704, 1.12373685836792, 0.5492866039276123, 0.5361461639404297, 1.4296233654022217, 1.0962285995483398, 0.45173725485801697], [2.1756176948547363, 0.4627268314361572, -1.0205833911895752, 0.1786973625421524, -0.6513407230377197, -1.266762137413025, 0.20114877820014954, 1.3650834560394287], [-1.5403521060943604, 1.4201704263687134, -0.12056515365839005, -0.28393542766571045, 0.0823911800980568, 0.409148633480072, 0.2425239086151123, 2.2739689350128174], [-1.1562937498092651, 1.2871911525726318, 0.3061583340167999, -0.3986823260784149, -0.30953747034072876, -0.22617857158184052, 0.19915427267551422, 0.5507158637046814], [1.0389156341552734, 1.6439738273620605, -0.7876169085502625, 0.15190552175045013, 1.2771356105804443, 0.5373921394348145, -0.9785148501396179, -0.0017445581033825874], [0.23093155026435852, 0.3644678592681885, -1.5569109916687012, -0.6021026372909546, 0.631466805934906, -0.24430331587791443, -1.6185697317123413, 1.0642062425613403], [1.2017159461975098, -1.0407859086990356, 0.6031026840209961, -1.0690221786499023, 0.7208902835845947, -0.38689160346984863, 0.9429667592048645, 0.3666425943374634], [-0.3800908625125885, 0.6269286870956421, -1.0195941925048828, 0.4058118462562561, -1.5697882175445557, 0.0015375185757875443, 0.14920783042907715, -1.0349419116973877], [-0.04487033933401108, 0.6434809565544128, -2.4999570846557617, 0.677003800868988, 0.265848845243454, -0.5996853113174438, -1.6005487442016602, -2.3663463592529297], [-0.059934113174676895, 0.7335631251335144, 0.5632102489471436, 2.179816246032715, 0.31255826354026794, -0.1954733282327652, -0.5787429213523865, -0.2821883261203766], [-0.34345167875289917, -0.2403782606124878, 0.5471432209014893, 0.9729377627372742, 0.3884202837944031, 0.03760450705885887, -1.4747295379638672, 0.7485731244087219], [0.5722677707672119, -0.2881016135215759, 0.6343083381652832, -0.7024849057197571, -0.5596035718917847, 0.2844959497451782, 1.9019732475280762, 0.13166247308254242], [1.2550538778305054, 1.4367579221725464, 1.3187702894210815, 1.305092215538025, 0.2850261330604553, -0.33370912075042725, -2.320863723754883, 0.38003140687942505], [-0.05517003312706947, -1.302726149559021, -1.172308325767517, 0.111366868019104, 1.044037938117981, 0.21177029609680176, 0.21619686484336853, -0.23652632534503937], [-0.008870182558894157, 1.0188004970550537, 1.297404170036316, -0.018862372264266014, -1.1051921844482422, 0.4169082045555115, 1.6548436880111694, -0.1991974413394928], [1.1770151853561401, 0.9078202247619629, -2.7293307781219482, 0.5852043032646179, -2.671992540359497, 0.9033976793289185, 1.426221251487732, -0.027593769133090973], [-0.0918271392583847, -0.43212372064590454, 1.9965660572052002, 0.443410187959671, -0.1928330361843109, -0.05780297890305519, -0.7733646035194397, -0.17574764788150787], [1.1585015058517456, 0.5599997639656067, 0.8081073760986328, 1.1470012664794922, 0.1922348290681839, -1.6387085914611816, -0.012152730487287045, 0.38095688819885254], [1.1058849096298218, -0.07777288556098938, 0.5385866761207581, 1.7591463327407837, -0.07611145079135895, -0.3118213713169098, 0.36160534620285034, 1.8156243562698364], [1.1356639862060547, 0.608169674873352, -0.5087509751319885, -1.1113193035125732, -0.4441082179546356, 1.3663578033447266, -1.5733333826065063, -0.2865455746650696], [0.8097206354141235, -0.9723192453384399, 0.9131670594215393, 0.24272865056991577, 1.1205520629882812, 0.1934967190027237, -0.19868850708007812, 0.9171271920204163], [-0.531065821647644, 0.5385504961013794, -0.9952021241188049, 1.539199948310852, 0.88228440284729, -0.12281034886837006, 0.9050129055976868, 1.2403110265731812], [0.43423935770988464, 0.1971718817949295, 0.00036142015596851707, -0.43383076786994934, -1.488384485244751, 0.6488706469535828, -1.9746193885803223, -2.4470484256744385], [0.8717919588088989, 0.2111101597547531, 0.682684063911438, 0.8930428624153137, 0.7428656816482544, 0.6685258746147156, 1.1701891422271729, 0.5916616320610046], [0.4825485944747925, 0.0038714739494025707, 0.5046852827072144, 1.728275179862976, 0.2610824704170227, -0.5867379903793335, -0.6585408449172974, -0.7162305116653442], [-0.17388971149921417, -1.0037983655929565, 0.6527949571609497, -0.542175829410553, -1.1497865915298462, -0.8873955607414246, 0.5171854496002197, 2.1403815746307373], [-0.4696541130542755, 0.4154439866542816, 1.3450067043304443, 0.8683132529258728, -0.19992497563362122, -1.1034307479858398, -1.4871761798858643, -0.24214385449886322], [-0.8360080122947693, 1.1588895320892334, 1.146791934967041, -0.8149833083152771, 1.3403327465057373, 1.1463277339935303, 0.6365511417388916, 0.9562512040138245], [-0.15135276317596436, -0.6172360777854919, 1.0034393072128296, 0.5162315964698792, 0.3328186869621277, 1.5544158220291138, -0.09668939560651779, 0.21471546590328217], [0.39753058552742004, 0.1001194640994072, -1.943913221359253, 0.5338232517242432, -0.08524986356496811, 0.11774526536464691, 0.1989464908838272, 0.5906761288642883], [0.5258367657661438, 0.7116095423698425, -0.2735048234462738, 0.22151459753513336, 0.8060279488563538, -1.319542407989502, 0.17849883437156677, -1.0967563390731812], [0.9877346754074097, 0.7614672780036926, -0.14779698848724365, 1.5538924932479858, -1.202980875968933, 0.021957462653517723, -1.0591585636138916, 0.8581293225288391], [-0.08022507280111313, 0.7608322501182556, 0.0658842995762825, -0.1984647512435913, -1.166196584701538, 0.7276686429977417, -0.7934562563896179, -0.9911741614341736], [-0.733374834060669, 0.7983412742614746, 1.438063383102417, 0.6120776534080505, 0.769134521484375, 1.5812698602676392, 0.32875683903694153, -0.6016169786453247], [0.5200664401054382, 1.2855756282806396, 0.2790091335773468, 2.15771484375, -0.7839892506599426, -0.2835506796836853, -0.8637877702713013, 0.10491256415843964], [-0.1543884426355362, -0.41731420159339905, -0.44840574264526367, 0.04914535582065582, 0.6500530242919922, 1.3540292978286743, -0.5581386685371399, 0.035793889313936234], [0.7217614054679871, -0.6051105260848999, -0.7499956488609314, -2.546915054321289, 0.3772425055503845, 0.7456338405609131, 0.281658798456192, -1.1780381202697754], [0.4324873387813568, 0.28891873359680176, -0.6904506087303162, 1.0138943195343018, 0.18214601278305054, 0.651258111000061, -2.093937635421753, 0.030770281329751015], [-0.17847713828086853, -0.2672005593776703, -0.3233930766582489, -0.6053787469863892, -0.3442949950695038, 2.027409315109253, -1.0892404317855835, 0.5066009163856506], [-1.0055253505706787, 1.6120152473449707, -0.588987410068512, -0.404759019613266, -0.03068680316209793, -1.1639108657836914, 1.1856544017791748, 0.2865322530269623], [1.0993951559066772, 1.5405515432357788, 0.9194950461387634, 0.050869088619947433, 2.0123496055603027, -0.7230669856071472, -0.29804009199142456, 0.5410785675048828], [0.5808771848678589, 0.3846074342727661, -0.868696928024292, 0.06740410625934601, 1.2313517332077026, 0.8531168699264526, 0.4396909177303314, 0.10529863089323044], [-0.31843438744544983, 0.6375857591629028, 1.6476671695709229, 1.2894887924194336, -0.6080324053764343, -0.16736756265163422, -0.06303451210260391, -1.3588862419128418], [0.38816583156585693, 0.6020118594169617, -0.9751248359680176, -1.3586227893829346, 1.0056878328323364, 0.2667406499385834, -0.2418149709701538, 1.9332412481307983], [-1.268805742263794, 1.0428493022918701, 0.7391679286956787, 0.18818046152591705, -0.015328599140048027, -0.6391578316688538, 1.2932956218719482, 2.326563835144043], [1.0635360479354858, 0.022759882733225822, -0.32243841886520386, -1.0202274322509766, 0.4398305416107178, -1.0905219316482544, 0.9628370404243469, 0.5326675772666931], [0.302954763174057, 1.3073674440383911, -3.394202709197998, 2.5207250118255615, 0.29956430196762085, -1.1753606796264648, 0.21522553265094757, 0.34950026869773865], [0.9278119206428528, -2.0446197986602783, -0.5584334135055542, 2.742349624633789, -0.534080445766449, 0.2925652861595154, -0.51056307554245, 0.5871586799621582], [-0.5823162198066711, -1.4247620105743408, 2.099827527999878, 0.23575849831104279, 0.20480726659297943, -0.37787172198295593, 0.3232809603214264, 0.5798158049583435], [1.4650219678878784, -0.9052669405937195, 1.3054064512252808, 1.006973385810852, -0.021978115662932396, -1.0648140907287598, -0.30682459473609924, -0.0950460210442543], [1.1611686944961548, -0.14057068526744843, -0.9369905591011047, -0.083205945789814, 2.0371408462524414, -0.14993704855442047, 0.05704266577959061, 1.052337646484375], [0.3698345422744751, -0.007714959792792797, 0.4793162941932678, 0.4051300585269928, -1.4641274213790894, -1.2521153688430786, -1.6600048542022705, 1.2652033567428589], [1.2623659372329712, -0.9227244257926941, -0.36405304074287415, -0.9239353537559509, -0.053916361182928085, 0.14731110632419586, 2.015169143676758, -0.05698753520846367], [1.8805795907974243, -1.6697794198989868, 1.1091121435165405, -0.812684178352356, -0.5053858160972595, 0.17367807030677795, -2.315213441848755, -0.5902934670448303], [0.7967305183410645, 1.935174822807312, -0.9060985445976257, -0.89872807264328, 0.2564641237258911, 0.40984323620796204, 0.29172518849372864, -0.03406071290373802], [0.5949801802635193, -0.44074228405952454, -1.5856518745422363, 0.47202619910240173, 0.5207944512367249, 1.3510048389434814, -2.100154399871826, -0.6395897269248962], [0.7420458197593689, -0.3317152261734009, -0.3485700488090515, -0.5228983759880066, 2.063624858856201, 0.06011960655450821, -1.2197511196136475, 0.9943854808807373], [0.9476432800292969, -1.627307415008545, 1.12163245677948, -0.2019922435283661, 1.737622618675232, 0.19366125762462616, -1.190651774406433, -0.2166559100151062], [-2.6225059032440186, -0.4670325219631195, 0.14113688468933105, -0.2570996880531311, -1.0347968339920044, 0.6148632168769836, -0.37971973419189453, -0.5748671293258667], [-1.8876285552978516, 1.258285641670227, -0.21683575212955475, -2.213404893875122, -0.8439642786979675, 0.9263184070587158, 0.7234667539596558, 0.6659275889396667], [1.8478150367736816, 0.6525042653083801, -0.3566541075706482, 0.7860165238380432, 0.20089761912822723, -0.9426977634429932, -0.9272936582565308, 0.7146748304367065], [-0.6360394954681396, 1.3740203380584717, 0.17942190170288086, 1.4656604528427124, -0.32943591475486755, 2.6883373260498047, -0.5019999146461487, 1.7937180995941162], [-0.5589315295219421, -1.3702690601348877, 1.108992338180542, 2.2730464935302734, 0.8596386313438416, 0.5448567271232605, 0.24280773103237152, -1.066595196723938], [-1.041121244430542, 0.25569725036621094, 0.008180678822100163, 0.211593359708786, 0.27637529373168945, 0.7764255404472351, 0.3200867772102356, -0.4931078553199768], [-1.4634042978286743, -0.3425239622592926, -0.4282916486263275, 1.474362850189209, 0.365707129240036, -0.5865411758422852, 0.2448156625032425, -0.1747400462627411], [-1.663293719291687, -0.042792994529008865, 0.9456743597984314, -0.5638426542282104, -0.8623204231262207, -0.5851919054985046, -0.829062819480896, -0.5767837166786194], [0.48175203800201416, 0.7933630347251892, 1.4007059335708618, 0.6041545867919922, -0.19055506587028503, 0.8371819257736206, 1.251096248626709, 0.24264827370643616], [0.6573781371116638, -2.0956673622131348, -0.10822713375091553, -0.6414809226989746, 0.5977939367294312, -0.5000418424606323, 0.45504406094551086, -0.20641635358333588], [-0.3778601884841919, -0.3129327893257141, -0.24237003922462463, -0.7330251336097717, -0.9843015670776367, -0.026386093348264694, 0.23774085938930511, 0.6784436702728271], [-0.4793015122413635, 1.570681095123291, 0.06497981399297714, -0.37245336174964905, -0.31350046396255493, -0.37706637382507324, 1.533678650856018, -0.13555513322353363], [-1.6873832941055298, 0.7599238157272339, 1.1774718761444092, -0.595248281955719, 0.12502625584602356, 0.8578765392303467, -0.31731241941452026, -0.7894783020019531], [-0.6100114583969116, 2.107875108718872, -0.19303058087825775, 0.4187330901622772, -0.7404711246490479, 1.4753023386001587, -1.4296917915344238, -1.2618013620376587], [-1.618111252784729, -0.46408337354660034, -0.4912499189376831, -1.0452566146850586, -0.22015714645385742, 0.6871035695075989, 0.9514214396476746, 0.9149875044822693], [0.08315575867891312, 1.0965501070022583, -0.2743087708950043, 1.1639938354492188, -2.769876003265381, 0.5625745058059692, -0.16317886114120483, -1.056869387626648], [-1.2913058996200562, 0.7502613067626953, 0.3055630028247833, -1.7620376348495483, 0.8168143630027771, 0.7723711729049683, -1.4717525243759155, -0.746867299079895], [-0.8711653351783752, 1.1101149320602417, 0.7828549742698669, 0.9714459180831909, -0.8983019590377808, 1.0161678791046143, -0.5573601722717285, 0.382241427898407], [-0.8280491828918457, 0.0024933938402682543, -0.6914898753166199, 1.096976637840271, 1.901038646697998, -0.30482369661331177, -0.4181496500968933, -0.3108041286468506], [-0.48648199439048767, -1.060141682624817, 1.7484230995178223, 0.759529173374176, 1.889359474182129, 0.07128193229436874, 1.1860926151275635, 0.713435173034668], [0.15999749302864075, -0.8494007587432861, -0.4944804906845093, -0.9338062405586243, 0.06855005025863647, 1.3365627527236938, -0.08371905237436295, -1.0037505626678467], [1.4103268384933472, 0.04293721541762352, -0.9317368865013123, 1.984667420387268, 0.006022780202329159, -0.8240267038345337, -0.18770824372768402, -0.7871174216270447], [0.6320507526397705, -0.7915630340576172, -0.9615688323974609, 0.8913049697875977, 0.5583609342575073, 1.1255544424057007, 1.7701977491378784, 0.42644423246383667], [1.023686408996582, 1.255289912223816, 0.11709656566381454, 0.8365507125854492, 0.6054449677467346, -0.9390354156494141, 0.31269732117652893, 0.5342097282409668], [-0.5443100333213806, 0.7418738603591919, 0.08316964656114578, -0.02209213748574257, 0.7111260294914246, -0.0952715203166008, 2.016998291015625, -1.2549580335617065], [0.4917491674423218, -0.9876230359077454, -1.502798318862915, -0.20094184577465057, 0.5836751461029053, -0.9646257162094116, -1.6841163635253906, 0.66005939245224], [2.10783314704895, 0.8346686363220215, -0.4440714418888092, -0.31334543228149414, -0.6339675188064575, 0.7838153839111328, -1.1942344903945923, 0.3822808265686035], [0.15193423628807068, -0.5012284517288208, 1.5148106813430786, 0.40522128343582153, 0.20867976546287537, 1.115073800086975, -1.2724623680114746, 0.4132859706878662], [-1.655066728591919, -1.89699125289917, -1.172831654548645, 1.0787622928619385, -0.5330622792243958, 1.046690583229065, -0.19249063730239868, 2.071039915084839], [-0.7751089334487915, -0.5337023138999939, -0.3675820529460907, -0.7466557621955872, -0.10137190669775009, 1.8736714124679565, -1.4864782094955444, -0.04053080826997757], [-0.07245989143848419, 0.945717990398407, -0.965747594833374, 0.928187906742096, -0.6743333339691162, -0.23624837398529053, -0.16012713313102722, -1.0104893445968628], [-0.23408079147338867, 0.6446107029914856, 0.8588594794273376, -1.0569233894348145, 0.42638421058654785, -0.05507994443178177, 0.904840886592865, 1.8576030731201172], [0.2823214828968048, -0.4661914110183716, 0.40723854303359985, 0.40312835574150085, 0.524960994720459, 0.5749394297599792, 0.06391917914152145, -0.30497029423713684], [0.4742957651615143, -0.13350529968738556, 0.1713293343782425, 2.1129636764526367, 1.1280525922775269, -0.2705356478691101, 0.43243861198425293, -0.1960608810186386], [-0.5624713897705078, -1.0770760774612427, 0.4526909589767456, -0.27053019404411316, 0.4830087721347809, -1.0108155012130737, 1.3954182863235474, -0.42499321699142456], [0.28493034839630127, -0.00975255761295557, -3.1386258602142334, 0.7100043296813965, -0.4366585910320282, -0.8916860222816467, 0.47105276584625244, -1.1563912630081177], [0.17590945959091187, 0.5567935109138489, 0.9461489319801331, 1.1150518655776978, -0.36686110496520996, -0.4793398082256317, 0.8078529238700867, -1.1731361150741577], [0.49700427055358887, 0.5291484594345093, -1.7023546695709229, 0.18876294791698456, 1.1683732271194458, 0.03527018427848816, 0.483220636844635, -0.19168895483016968], [1.0852686166763306, -1.3161894083023071, -2.6186442375183105, -0.7064722776412964, -0.003924536053091288, 0.4723483622074127, -2.2046573162078857, -0.6497318148612976], [-0.5235285758972168, 0.9066085815429688, 0.595594584941864, 0.49905455112457275, 2.421298027038574, 2.2365036010742188, 1.1857613325119019, -2.1883888244628906], [-0.5784037113189697, -2.1066951751708984, -0.5456036329269409, 0.332985520362854, -1.063854455947876, 0.9878848195075989, -0.19354620575904846, 1.0338363647460938], [0.14083389937877655, -1.084449291229248, 1.3089478015899658, -1.4857406616210938, -1.1338002681732178, 1.0683671236038208, -0.15525370836257935, 0.30824923515319824], [-0.3030955493450165, -0.6937339305877686, -1.5314764976501465, 0.7356353998184204, 1.017983078956604, -0.16437898576259613, -0.2947652339935303, 0.8849761486053467], [-1.4962836503982544, -0.34484025835990906, 1.5473524332046509, 0.22755762934684753, 1.2658377885818481, -0.20060406625270844, -0.4883538484573364, 0.08468108624219894], [-0.061078671365976334, -0.03952151536941528, 0.5865839719772339, 1.3096407651901245, -1.2593461275100708, -0.38947781920433044, -0.2662402093410492, -0.29915937781333923], [0.9157175421714783, -0.23458197712898254, -1.1578396558761597, -0.8514638543128967, -0.7381569743156433, -0.8954095840454102, -0.3946029841899872, -3.1403026580810547], [0.2583799958229065, 0.7562066912651062, 0.16438913345336914, 0.2981015145778656, -1.6673952341079712, 0.18542547523975372, -0.048007782548666, -0.10243069380521774], [-1.919180989265442, -0.720258891582489, 0.307950884103775, 1.927223563194275, -0.010534228757023811, 1.0443642139434814, -1.7846087217330933, 0.2498432695865631], [0.9614304900169373, 1.3232532739639282, 1.3720897436141968, -0.6112096905708313, 0.3756261467933655, -0.11222165077924728, 0.14613328874111176, 0.6103403568267822], [0.4941205680370331, -0.17085886001586914, -0.8152959942817688, -1.5080974102020264, -1.5356944799423218, 1.305529236793518, 0.20977438986301422, 0.5075126886367798], [1.7787772417068481, 0.6363085508346558, -0.7889668345451355, -0.8526017069816589, 0.6059049367904663, 1.0575505495071411, -0.31012803316116333, -0.11205029487609863], [0.8078479766845703, 1.3068945407867432, 0.9218181371688843, -0.6725379824638367, 0.09025917202234268, 0.3132092356681824, 0.7385586500167847, -1.1765968799591064], [-0.3101919889450073, 0.14980728924274445, -1.6521658897399902, -2.1475977897644043, -0.18916040658950806, -0.16630378365516663, 0.956203281879425, -0.1296168714761734], [0.20321644842624664, 0.37574872374534607, -0.23841769993305206, -0.1572147160768509, 0.1960703730583191, -1.9776499271392822, 1.438205361366272, 0.0012074782280251384], [0.09836617112159729, -0.595453679561615, -0.9791212677955627, -0.8589543700218201, 0.19791406393051147, -0.06761375069618225, -0.8725399971008301, 0.7402382493019104], [-1.5374723672866821, 0.3732011616230011, -0.3228554129600525, 0.8737740516662598, -1.5618455410003662, -0.6848052740097046, -0.675510048866272, -1.636957049369812], [-1.4422551393508911, -0.0961538627743721, -0.41095295548439026, -2.212986469268799, -0.312085896730423, 2.272712469100952, 0.0756411999464035, 1.0460827350616455], [-1.665600299835205, 0.5721181631088257, 0.28655800223350525, 0.5814282298088074, -0.8719560503959656, -0.7616409063339233, -0.5560441613197327, 0.7666398882865906], [-0.34932106733322144, 0.8649181127548218, -0.29744285345077515, -0.7349597811698914, 0.9591972827911377, -0.3325442969799042, 1.2363667488098145, 0.33000820875167847], [0.45697152614593506, -0.5952625870704651, -0.2619260549545288, -1.752129316329956, -0.1160571277141571, 0.2909087538719177, 0.2827187776565552, 0.8610588908195496], [-0.3827415406703949, -1.4669767618179321, 0.1021459624171257, -0.5438435673713684, -1.2093448638916016, -0.01586362160742283, -0.24499933421611786, 0.3126993179321289], [0.8908519148826599, 1.5670970678329468, 0.5488126277923584, -0.28228959441185, 0.8641465902328491, -1.6279151439666748, 0.7342387437820435, -1.528693437576294], [-1.0548125505447388, -0.021733732894062996, 0.1566700041294098, -1.1017314195632935, 1.2944247722625732, -0.3118622899055481, 0.19403432309627533, 0.9408056735992432], [0.8073091506958008, 1.1432886123657227, 1.1631286144256592, -0.9111976027488708, 1.146109700202942, 0.1632513850927353, -0.6055102348327637, -1.8311458826065063], [-0.49106937646865845, -0.03720353543758392, 0.5466578006744385, 0.5858879685401917, 0.6363037824630737, 0.2453085035085678, -0.7411762475967407, -1.147349238395691], [-0.6328967809677124, -0.5411592721939087, -0.11442532390356064, -0.00821747351437807, -1.9080482721328735, -0.26200011372566223, 0.41197794675827026, 0.4470808506011963], [0.1720125377178192, -0.6952779293060303, -0.6881990432739258, 0.3364565968513489, 1.3107465505599976, 0.15772579610347748, 0.12965957820415497, -1.1763052940368652], [0.41218432784080505, 1.8174161911010742, 1.398464560508728, 1.3172141313552856, -1.5016446113586426, -0.26599958539009094, 0.09791883081197739, -0.6256298422813416], [-0.11431178450584412, -1.8438090085983276, 1.92145836353302, 0.26069849729537964, -1.338067889213562, 0.2393634170293808, -0.21449661254882812, -1.730669617652893], [-0.3808411657810211, -0.1919376254081726, 0.8112621903419495, 0.5868936777114868, 0.6727450489997864, -0.4863080680370331, 1.8079684972763062, 0.8188982605934143], [0.09947226196527481, -1.1842656135559082, -0.64417964220047, 0.4756954610347748, 1.0714839696884155, -0.3440982699394226, -1.805983543395996, -1.043518304824829], [-0.37432804703712463, 0.991097629070282, -0.9247588515281677, 2.3390989303588867, 0.17707884311676025, -0.5766146183013916, -0.5861037969589233, -0.5769453048706055], [-0.25785139203071594, 0.28945839405059814, 0.4682147204875946, -1.1631823778152466, -0.5358993411064148, 2.065391778945923, -0.6894469857215881, 0.044631801545619965], [0.11398189514875412, -0.59053635597229, 0.9177821278572083, -0.3475203812122345, -0.696180522441864, -1.2717012166976929, 0.5120403170585632, 0.7425954341888428], [1.0720875263214111, -0.022792581468820572, -1.2609597444534302, -0.7251799702644348, -1.2118005752563477, 0.6173693537712097, -1.2884844541549683, -1.6042144298553467], [-1.1596168279647827, -0.32216617465019226, -0.8090204000473022, -0.11138202250003815, 1.0557665824890137, -0.1636064201593399, -0.9659748077392578, 0.7130533456802368], [-0.8366473913192749, 0.5589631795883179, 0.34508833289146423, 0.6614177823066711, -0.23407700657844543, -0.1479426920413971, 0.7339503765106201, -0.48446738719940186], [1.1932154893875122, -0.007550982292741537, -0.9822742938995361, -0.7258617877960205, -0.18919925391674042, 0.15271686017513275, -0.7164987325668335, -0.3358219861984253], [-0.5581791400909424, -0.9754022359848022, 1.4233081340789795, 0.11537430435419083, -0.7277413606643677, 0.2075759470462799, 0.7494686245918274, -1.5661972761154175], [0.20261242985725403, 1.0224494934082031, 0.7260743379592896, 1.2075510025024414, -0.17405912280082703, 1.0286953449249268, 1.4560712575912476, -0.32288575172424316], [0.3555263578891754, 0.504655122756958, 0.08284103125333786, 0.04300881549715996, -0.023054860532283783, 0.18634390830993652, -1.3143842220306396, 1.517643928527832], [-1.3549704551696777, -0.9992358684539795, 0.1637558937072754, -1.9660969972610474, 0.3385125696659088, 1.666174054145813, -0.7404653429985046, 0.7269906401634216], [-2.006492853164673, 1.9879158735275269, 0.34713229537010193, -2.0641865730285645, 1.651599645614624, -1.2340670824050903, 0.4110057055950165, -1.5639842748641968], [-0.32133349776268005, 0.7741835117340088, -0.7784367799758911, -1.2565194368362427, 2.4436802864074707, 2.8115475177764893, -1.636869192123413, -1.1004408597946167], [-0.8306759595870972, -0.6296494007110596, -1.7297358512878418, 0.8821850419044495, 0.6135115027427673, -1.0671215057373047, -0.12899456918239594, -0.8345120549201965], [1.138739824295044, -0.8616366982460022, -1.183185338973999, 0.4657377302646637, -2.0230157375335693, -0.22653627395629883, -0.8043884038925171, 0.342245876789093], [-0.7356572151184082, -1.1566286087036133, -0.4630025625228882, -1.6423966884613037, 0.916099488735199, -1.3085248470306396, 0.6065971851348877, 1.26985764503479], [-0.13340969383716583, -0.4670799970626831, 0.03244781866669655, -1.312052845954895, 1.0431381464004517, -1.2044036388397217, -1.6600815057754517, -0.5698022246360779], [-0.306744784116745, -0.6063827872276306, -0.34577512741088867, -1.5941014289855957, 1.4315528869628906, -1.0785547494888306, -0.43934348225593567, -0.2862241864204407], [-0.33192265033721924, 0.2753937542438507, 1.4851411581039429, -1.8122739791870117, -0.7096453905105591, -0.81524258852005, 0.3542404770851135, 0.8802616596221924], [-0.4560236930847168, -0.15480774641036987, 1.438897728919983, 0.9247260689735413, 0.7801960706710815, -1.1688355207443237, 0.6362119317054749, 0.20769861340522766], [0.6318837404251099, -0.4854954481124878, -0.20626141130924225, -1.3270342350006104, -2.1100287437438965, -0.17826266586780548, 0.4096870422363281, -0.22571049630641937], [-1.4461325407028198, 1.7094272375106812, 0.7868101596832275, 2.113626718521118, -0.23297622799873352, 0.8402455449104309, 0.22518810629844666, -1.5335890054702759], [-0.8230178356170654, 0.5464675426483154, -0.9222293496131897, 1.9192668199539185, -1.2965914011001587, -0.39363953471183777, -1.2039422988891602, -1.0545777082443237], [1.292222261428833, 1.4084550142288208, 1.6378834247589111, 0.4890323281288147, -0.3176746368408203, 0.16463249921798706, -1.2718448638916016, -0.3159181773662567], [0.2554721236228943, -0.40582185983657837, 2.0454225540161133, -0.06490227580070496, 0.16257208585739136, 0.5406899452209473, 1.4378762245178223, 0.08287665247917175], [0.5458349585533142, -0.9316844344139099, -1.129281997680664, -0.4525051712989807, 0.6924667358398438, -1.1627390384674072, -1.6302058696746826, 1.5794451236724854], [0.8321049809455872, 1.331910490989685, -1.2196204662322998, -1.1902351379394531, -0.2638758420944214, 1.0998026132583618, -2.2447075843811035, 1.9849376678466797], [-0.25602638721466064, 1.4106998443603516, 1.7273688316345215, 0.5756895542144775, 1.0531244277954102, -0.9648179411888123, -0.6339163780212402, -0.7942713499069214], [-0.4585694372653961, 1.2606570720672607, 0.11341787874698639, 1.2972806692123413, 0.09587237983942032, -0.9954245686531067, -0.8359984755516052, 2.7181310653686523], [0.691288948059082, 0.14850273728370667, -0.6003252267837524, -0.2099764198064804, 0.0870722308754921, 0.6105910539627075, 0.04639236629009247, 1.5970405340194702], [-1.1416120529174805, -0.027103746309876442, 2.0577569007873535, -0.6085034608840942, 0.39313462376594543, 0.9600736498832703, -0.08530674129724503, -0.7470259070396423], [-0.8237984776496887, -0.4119020700454712, 1.7185089588165283, 0.05002404376864433, 0.7194754481315613, 0.897956132888794, -0.09780064225196838, -0.26074469089508057], [0.5424817800521851, 0.9853000044822693, -1.0299394130706787, -0.5707763433456421, -0.652305006980896, -1.7753665447235107, -0.7948888540267944, 0.009770860895514488], [-1.6139036417007446, -0.5349546074867249, 0.9748477935791016, 0.006227550096809864, -0.6395313739776611, 0.9203718900680542, 0.06935491412878036, 0.30810800194740295], [-0.1277734488248825, 0.6199594140052795, 0.40001699328422546, 0.012211873196065426, 0.3576899766921997, 0.09472494572401047, -0.6875860691070557, -0.14035020768642426], [0.4901241064071655, -0.8873517513275146, 1.041859745979309, -0.11208728700876236, 0.12932635843753815, 0.992122232913971, -0.44640615582466125, 1.2369827032089233], [0.49255073070526123, -0.08863583207130432, -1.1171491146087646, 0.7433570623397827, -0.25553447008132935, 0.7011019587516785, 1.0461422204971313, -0.08598306030035019], [0.7029294967651367, 0.40068960189819336, 0.6719003915786743, -3.0847535133361816, 0.7234988808631897, 0.44738468527793884, -1.1847280263900757, 1.1820690631866455], [-1.8060169219970703, -0.2658676505088806, -1.2712492942810059, 0.43944600224494934, -0.2915353775024414, -0.26028183102607727, -0.04186311736702919, -0.07956022024154663], [0.29772692918777466, 0.7749704122543335, -1.773538589477539, 0.029972679913043976, 0.22304393351078033, 1.2940244674682617, 2.0680601596832275, 1.664398193359375], [-0.7255547046661377, -0.312203586101532, -0.45233815908432007, 2.8254199028015137, -0.7646694779396057, -0.05940936133265495, 1.0825786590576172, 0.15378710627555847], [1.2855356931686401, 0.2730717360973358, -1.002369999885559, -1.16218900680542, -0.13216465711593628, -0.2427709400653839, -0.2812114953994751, 0.8992035984992981], [0.545623242855072, 0.3510449528694153, -2.32690167427063, -0.47931182384490967, -0.8492889404296875, 0.309826523065567, -0.29613110423088074, 0.41832107305526733], [1.460415244102478, -1.7333780527114868, 0.3601468801498413, -1.3294578790664673, -0.16180916130542755, 0.34157341718673706, -0.6821791529655457, -0.4798237085342407], [0.12818631529808044, -0.17849141359329224, -2.1152637004852295, 0.818642258644104, -1.4338419437408447, -0.9073707461357117, -0.9147263169288635, -0.9373217225074768], [0.6136392951011658, -0.41137823462486267, -0.6372761130332947, 0.7796814441680908, 1.2052313089370728, -0.3415462076663971, 1.440528392791748, 0.5357574224472046], [-0.07444748282432556, 0.39773139357566833, 0.11996039003133774, 0.4123609662055969, -0.06611400097608566, -0.6730955839157104, -0.42704781889915466, 0.07951797544956207], [1.4063438177108765, 0.601302981376648, -1.8309686183929443, -0.4139494299888611, 0.5519109964370728, -0.7986673712730408, 0.10241927951574326, -0.3900836110115051], [0.15340641140937805, -0.15181700885295868, -0.4463135600090027, 0.339009553194046, -0.7732347249984741, 0.4677644670009613, -0.33371689915657043, -0.3094026744365692], [-0.26222461462020874, -2.308138847351074, 0.8387052416801453, -0.04314319044351578, 1.2718433141708374, 0.19245053827762604, -0.22683362662792206, 0.06207113340497017], [1.1841304302215576, 0.8202620148658752, 0.4822984039783478, 1.0980147123336792, 0.7001399993896484, 1.3301438093185425, -0.34202468395233154, -0.9383918046951294], [-0.0405607670545578, 1.8503565788269043, -1.287029504776001, -0.9048593044281006, 1.6380469799041748, 0.7171326279640198, 0.3786587119102478, -1.439864993095398], [1.0245513916015625, 2.3187663555145264, -0.4344286620616913, -0.9654132723808289, -0.19128882884979248, 0.9560630917549133, 1.437997579574585, -0.20077605545520782], [2.0635805130004883, 0.3419052064418793, -0.37274476885795593, -0.751790463924408, 0.14811348915100098, 0.006337046157568693, -1.5845974683761597, 0.8236908912658691], [-0.8004439473152161, -0.7885300517082214, 0.298335462808609, 0.6912591457366943, -0.5763977766036987, 0.7194003462791443, -0.9900454878807068, 0.6349846124649048], [-0.7698335647583008, 1.0707581043243408, -1.038602352142334, 2.225475788116455, -0.06511478871107101, -1.075219750404358, -1.28374445438385, -0.047585006803274155], [-0.4988502562046051, -0.08133772015571594, 3.1569838523864746, -0.1489943265914917, 0.8297939300537109, 0.23233187198638916, 0.3685198128223419, 0.41160884499549866], [-0.03106139414012432, 1.2893998622894287, -0.08112619072198868, -0.16458405554294586, 1.5147039890289307, -0.6077685356140137, 1.506649136543274, -0.6488917469978333], [-1.1402369737625122, 1.500435471534729, 0.3482094407081604, 0.459427148103714, 0.5385192036628723, 0.3974156975746155, 0.3954452872276306, -1.201621413230896], [1.097436785697937, -0.7101357579231262, -0.1713387370109558, -0.400438517332077, 0.7145797610282898, -0.49865809082984924, 1.180579423904419, -0.6277440190315247], [-0.03558177500963211, 0.09558682143688202, 1.218359112739563, -0.5402832627296448, 0.867805004119873, -1.6041642427444458, -1.5336867570877075, 0.7902398705482483], [-0.19477519392967224, 2.1980812549591064, -0.11305247992277145, -0.8696735501289368, 1.1953375339508057, 0.007401651237159967, 1.724730372428894, -0.7447795271873474], [2.699063539505005, -0.17015904188156128, 1.5568945407867432, 0.6746405959129333, 0.3964560329914093, 0.10002709180116653, -0.3665274679660797, -0.4682430326938629], [0.32386675477027893, 1.0822701454162598, 0.2596310079097748, 0.40764033794403076, -0.049676794558763504, -1.1375236511230469, -0.1491595059633255, 0.09916527569293976], [1.0094836950302124, -1.914393663406372, -0.37003231048583984, 1.5404210090637207, 0.046688444912433624, 1.3313415050506592, 0.43030139803886414, -0.2678976058959961], [0.4387443959712982, -0.14265750348567963, -1.6243712902069092, -1.3168296813964844, -0.011328499764204025, 0.4998035728931427, 1.0649584531784058, -0.184316948056221], [0.8046609163284302, 0.4289616048336029, -0.16021375358104706, -0.053583405911922455, -0.7782765030860901, -0.025512009859085083, -0.24628984928131104, -0.06292062252759933], [-0.2870056927204132, -0.7960376143455505, 0.3720491826534271, -0.7202090620994568, 2.367309331893921, 0.5466642379760742, -0.8931624889373779, -0.5960583090782166], [-2.175184726715088, 1.2549619674682617, 0.39269745349884033, -2.0801467895507812, -0.626944899559021, -1.2615435123443604, -1.398447871208191, -0.11697268486022949], [0.5207398533821106, -2.0185022354125977, -0.5800189971923828, -0.1009158045053482, -0.019910991191864014, -0.4445103704929352, -1.2674356698989868, 1.0075092315673828], [-2.299804210662842, 0.6789677143096924, -0.8047887682914734, -0.670913815498352, 0.4072611927986145, 0.33032625913619995, 0.7780881524085999, 0.9201725125312805], [2.0978705883026123, 1.3022514581680298, -0.020197639241814613, -0.6561828255653381, 0.40028518438339233, -0.5978018045425415, -0.46921882033348083, -0.4060700535774231], [0.048483435064554214, -1.0656934976577759, -0.10825926810503006, -0.1911032646894455, -0.7458849549293518, 0.5111226439476013, 0.09035731106996536, -0.23001334071159363], [-1.440220594406128, -0.47412070631980896, -0.6785915493965149, -0.784456729888916, 0.8304593563079834, 0.06885755062103271, 0.17731262743473053, -1.050285816192627], [0.2096930295228958, -1.5862462520599365, 0.39199960231781006, -0.5175211429595947, 1.6694940328598022, -0.8843429088592529, 0.0408150814473629, -1.0883771181106567], [-1.5298616886138916, 0.948392927646637, -1.128027319908142, 1.6171596050262451, -1.820120930671692, 0.6821619272232056, 1.300767183303833, -0.09241154789924622], [1.042873501777649, -0.30909091234207153, -1.3913406133651733, 0.4262188673019409, -1.0688450336456299, 0.15618683397769928, -0.34916651248931885, -0.4445316791534424], [0.9588321447372437, -0.40689244866371155, -0.6145384907722473, -0.7570763826370239, 1.9983566999435425, 1.7482744455337524, -1.0888276100158691, 0.9386390447616577], [0.5600239634513855, -0.4248059391975403, -0.9710782766342163, 0.03851454332470894, -0.43756726384162903, -0.7086721658706665, -1.132055640220642, 0.4101698100566864], [0.7476674318313599, 0.7017131447792053, 0.11643202602863312, 0.976026713848114, -0.5307645797729492, 0.3516165316104889, -0.3505072593688965, -0.9083892107009888], [-1.6068971157073975, -0.3321676552295685, -1.1598200798034668, -0.040590520948171616, -0.6355006694793701, -0.5888372659683228, 0.2010616958141327, 0.39730989933013916], [-1.046297550201416, -0.08901197463274002, -0.20442454516887665, -1.0668880939483643, -0.3928599953651428, -0.10488800704479218, -0.13934718072414398, -0.1555621176958084], [-1.833417534828186, 0.1791795790195465, -1.7292853593826294, 1.8616660833358765, -0.06897921115159988, -1.3632779121398926, 0.03658642619848251, -0.8087212443351746], [-3.147473096847534, 1.0772147178649902, -0.43840575218200684, 0.5888518691062927, -0.2917172610759735, -0.7117149829864502, 0.6272274255752563, 0.7626801133155823], [0.9638285636901855, -1.2211843729019165, 1.2422714233398438, -0.4070957601070404, -0.6317825317382812, 0.7025715112686157, -0.25903141498565674, 0.058794599026441574], [-0.9192467927932739, 0.14277416467666626, 0.764618992805481, 0.9494898319244385, 0.12380225956439972, -0.2832741439342499, -0.39166292548179626, -0.3742332458496094], [-1.0225539207458496, 0.14751067757606506, 1.2801917791366577, 0.44795364141464233, 0.2671652138233185, -0.1620539426803589, 0.606022298336029, -0.9674599170684814], [-1.521241307258606, 1.8602192401885986, -0.45074358582496643, 1.3440680503845215, 0.11032271385192871, -0.8142326474189758, -0.27784794569015503, -0.09855920821428299], [0.188307523727417, 1.6068476438522339, 1.6354295015335083, 0.5391663908958435, 1.2162495851516724, -0.8385022878646851, -1.0676915645599365, 0.8959577083587646], [-0.6770842671394348, -0.4971733093261719, -0.09502044320106506, 0.07002046704292297, 0.9535443782806396, 0.7867904305458069, -0.8824975490570068, 0.13559556007385254], [0.4736826419830322, 0.07415857166051865, -0.7466434836387634, -0.22266404330730438, -2.6330406665802, 0.08702229708433151, -0.14735610783100128, 0.5207609534263611], [1.5209598541259766, 0.5504130721092224, -2.073604106903076, 0.8952171206474304, 0.7223214507102966, -1.2227481603622437, -0.6167500019073486, 0.9796234369277954], [-0.8249156475067139, -1.0443059206008911, 2.4420626163482666, 1.6081640720367432, 0.7619712948799133, 0.36193111538887024, 1.0599490404129028, 0.221269428730011], [-0.20826290547847748, -0.3698378801345825, -0.4718593955039978, -0.3481191098690033, -0.051222432404756546, 0.14198991656303406, -1.100128173828125, 2.4171950817108154], [1.144474983215332, -0.20311623811721802, -2.439908981323242, -1.7781966924667358, 0.42296329140663147, -0.7681930065155029, -0.05638473480939865, 0.5503803491592407], [1.7619379758834839, 0.36233028769493103, 0.10195425152778625, -0.738109290599823, -0.6942726373672485, -0.44500136375427246, 0.31261977553367615, -0.3577215075492859], [-1.0015850067138672, 1.6862598657608032, 1.8591248989105225, 1.3350352048873901, -0.18784581124782562, 0.8471863865852356, -2.1206891536712646, 0.10183962434530258], [-0.7010325193405151, 0.953291118144989, 0.2482256144285202, 0.30492228269577026, 1.0381090641021729, -0.12684524059295654, -0.10067547857761383, 0.6399572491645813], [0.005969315301626921, -0.419735312461853, 0.2333606332540512, -0.9167275428771973, -1.0645521879196167, -0.10793567448854446, -1.1267499923706055, -0.8872592449188232], [0.4263460338115692, 0.9539874792098999, -0.6523800492286682, 1.6024527549743652, -1.7740302085876465, -1.2912110090255737, 0.5580291748046875, 1.6379613876342773], [-0.4208858907222748, -1.5337343215942383, -0.715497612953186, -0.36367884278297424, -1.122850775718689, -0.6626002788543701, 1.109212875366211, 1.4148354530334473], [-0.7444428205490112, 0.7191582322120667, 1.5782248973846436, 0.507032573223114, -0.5322193503379822, 1.5384306907653809, 1.1572169065475464, -0.6103771924972534], [-0.30545467138290405, 0.39818572998046875, 1.458125352859497, 0.9944756031036377, 1.3146635293960571, 0.7684144973754883, -1.2326226234436035, -2.242600440979004], [-0.7823355793952942, 0.48970383405685425, -0.33910492062568665, 0.4637788236141205, 0.12185171991586685, 0.21178190410137177, 0.08561569452285767, -0.8794935345649719], [1.4592136144638062, 0.8085546493530273, -0.1946694254875183, 1.262037754058838, -0.32587942481040955, -0.3426429033279419, 1.9310818910598755, -1.0868098735809326], [-0.48498955368995667, 1.2465062141418457, 0.08513465523719788, -1.5245416164398193, 0.343917578458786, -1.5686813592910767, 1.282713532447815, 1.179840087890625], [0.5937638878822327, 0.04065486043691635, -1.098413109779358, -0.6480870842933655, -0.20301827788352966, 0.38995614647865295, -0.04394896328449249, 0.6575700044631958], [0.33902814984321594, -0.7773433923721313, -1.0789343118667603, -0.4193056523799896, 1.9953303337097168, -0.764518141746521, -0.2071353942155838, -0.6894745826721191], [-0.05315147712826729, -0.6556375622749329, -0.46262606978416443, 1.120219111442566, 0.48484107851982117, -1.6815625429153442, -0.5141850113868713, 1.3962141275405884], [-1.113165020942688, -0.5859361290931702, -0.09717836230993271, -0.9411860704421997, -0.19504044950008392, -1.0609723329544067, -0.1609726995229721, 0.8815608620643616], [2.2742655277252197, 1.6395466327667236, -0.23139332234859467, 0.6038656830787659, -2.1800572872161865, -0.7845020890235901, -0.45836877822875977, -0.6534435749053955], [-0.2660176753997803, 1.6036735773086548, -0.03748257830739021, -0.2723778188228607, -1.95708167552948, -0.3593193590641022, -0.01029886119067669, 0.14252279698848724], [0.6344817280769348, 0.785182774066925, 0.5335347652435303, -0.40695616602897644, 1.60702645778656, -2.167888641357422, 0.10818994790315628, 1.8291057348251343], [-1.0905112028121948, 0.27516499161720276, -0.04616527631878853, 0.44093453884124756, 0.9744990468025208, -0.2968660891056061, 0.43588972091674805, -0.4073267877101898], [-1.3889025449752808, 0.03447149321436882, 0.9223219752311707, 0.21457412838935852, 0.24334768950939178, 1.6621217727661133, 0.352069616317749, -0.04781041666865349], [-0.2135835736989975, -0.4562373757362366, 1.5672028064727783, 0.6383653283119202, 0.8258607983589172, -0.7765404582023621, 0.7662233710289001, -0.3209962546825409], [-0.1939418762922287, 0.12914705276489258, 0.055396243929862976, -0.12489910423755646, -2.101576089859009, -0.3931981027126312, -0.9478120803833008, -1.1541870832443237], [0.9719904661178589, 1.3782916069030762, -0.3752460479736328, 0.8696995377540588, -0.33230510354042053, 1.6950852870941162, -0.5516993403434753, -1.2872190475463867], [-0.007072944659739733, -0.046438299119472504, -0.5706024765968323, -0.2257002294063568, -0.08336478471755981, 1.4058815240859985, -0.0027773831970989704, -0.7086669206619263], [0.268353670835495, -0.1934828907251358, -1.971518635749817, -0.13579800724983215, 0.2302086502313614, 0.16774916648864746, 0.4448641240596771, -1.2576202154159546], [-0.022902218624949455, 0.07293139398097992, -0.08928846567869186, 0.826302707195282, -0.2871442437171936, 0.0673593208193779, 0.2010609656572342, 2.0524682998657227], [-0.30550533533096313, -0.36721792817115784, -0.6029022336006165, -0.5823249816894531, 0.8607767820358276, -0.36718979477882385, 1.0404856204986572, -0.3421981632709503], [-1.402847170829773, -0.15442074835300446, -2.8386070728302, -0.11655072122812271, -0.30773812532424927, 1.1338768005371094, 0.15620584785938263, -1.4261194467544556], [0.3795566260814667, -1.2530378103256226, -0.07912182807922363, 0.37986770272254944, -0.44216907024383545, 0.2652237117290497, 0.1145089790225029, 0.08570022881031036], [1.1945257186889648, 1.245755910873413, -0.34970623254776, 0.7572149634361267, -1.9830440282821655, 0.2672467827796936, 0.5748947262763977, -0.5832275748252869], [0.10285864025354385, -0.1774589866399765, -0.6734539270401001, 1.9080169200897217, 1.9953672885894775, 1.1572223901748657, 1.354402780532837, -1.163049578666687], [-1.0736243724822998, 0.6160737872123718, -1.2752571105957031, -0.9423052072525024, -0.5568415522575378, -0.5312264561653137, -0.7399535179138184, 1.4473624229431152], [0.3371466100215912, -0.10374061018228531, 1.0857529640197754, -0.304988294839859, -1.087889552116394, 1.3855617046356201, 0.10639443248510361, -0.9030419588088989], [0.6534319519996643, 0.2640654444694519, -1.0017846822738647, 0.30412915349006653, -0.6010235548019409, -1.622377634048462, -0.22568224370479584, -0.4492805600166321], [-0.4948498010635376, -0.6191124320030212, -0.9719083905220032, 0.08846800774335861, 0.91487717628479, -0.9103933572769165, 0.03377305343747139, -1.0315085649490356], [-0.4207114279270172, 0.5132988691329956, 0.14282746613025665, 0.7704818844795227, 0.47054189443588257, -2.452376365661621, -1.6385899782180786, 0.16729256510734558], [-2.1007251739501953, 1.1699808835983276, 0.7836326956748962, 0.42913487553596497, 0.3928427994251251, -1.991791009902954, 0.28556016087532043, -0.9077772498130798], [-0.5264652371406555, 0.774246871471405, -0.14045080542564392, 0.7879547476768494, 0.1909785121679306, -0.09630318731069565, 0.7677037119865417, 1.053238034248352], [-0.9944824576377869, -1.3179682493209839, -0.013619625940918922, 0.07258100062608719, 0.5394963622093201, 0.44540008902549744, 0.9037668704986572, -1.612251877784729], [0.4720054566860199, -0.745570957660675, 0.704544186592102, -0.44373977184295654, 0.37248119711875916, 0.3329927623271942, 0.4953997731208801, -0.3234691023826599], [-1.364949107170105, 2.131629228591919, 0.5842700600624084, -1.0604480504989624, -0.6845114231109619, 1.2039315700531006, -0.005602188408374786, -0.2855920195579529], [-0.621773898601532, -0.40131157636642456, -0.33125147223472595, 1.3374743461608887, 0.5741623640060425, -0.3799022138118744, -1.1834698915481567, -0.5225549340248108], [0.5327121615409851, -0.8798890113830566, -0.015487141907215118, -1.1646956205368042, -1.2590488195419312, 0.6767770051956177, 2.0943658351898193, -0.4020889103412628], [-0.21593202650547028, -0.15620838105678558, -0.7337930202484131, 0.13911771774291992, -0.34449303150177, 0.892150342464447, 2.270145893096924, -1.0047818422317505], [0.3108249008655548, -0.20116448402404785, 0.29697248339653015, 0.8072236776351929, -0.27434980869293213, -0.4969254434108734, 0.053735241293907166, 0.21130135655403137], [1.0514981746673584, 0.2635907232761383, -0.07803145796060562, -0.8743468523025513, 0.03364339470863342, 0.34137749671936035, 0.8116734623908997, -1.827630877494812], [-0.7756872177124023, 0.5496753454208374, -0.9629698395729065, 0.31690514087677, 0.3762161135673523, -0.9062685370445251, 0.3296528160572052, -1.4425772428512573], [0.0195330660790205, 1.1058149337768555, -1.1852402687072754, 1.3793041706085205, -0.16487827897071838, -0.2734980881214142, 0.05446967855095863, 0.7537052035331726], [0.8197842836380005, -0.3933603763580322, -0.4968329071998596, 1.2390559911727905, -0.8615432381629944, -1.0188316106796265, -0.15087169408798218, -1.2106575965881348], [-0.6608087420463562, -0.6104697585105896, -1.65215003490448, -1.094225525856018, -0.6468597054481506, 0.6826081871986389, 0.03649575635790825, 1.4235692024230957], [-1.540556788444519, -0.9407926797866821, 0.6209815740585327, -0.525564968585968, -0.2206496298313141, 0.7736586928367615, -0.681481122970581, -2.189927816390991], [0.45697417855262756, 2.6414895057678223, 0.11642199754714966, -0.18257103860378265, -0.19742102921009064, -0.4728357791900635, 0.3559921383857727, -0.07406509667634964], [-1.1035715341567993, -0.9959363341331482, -1.0542175769805908, 0.27219516038894653, -0.048330482095479965, -1.343727469444275, -0.8510695099830627, 0.5826545357704163], [-0.7031744122505188, 0.3833511471748352, -1.5851037502288818, 0.4095149338245392, -0.3617694675922394, 1.14878249168396, -1.3125234842300415, -0.572312593460083], [0.5846042037010193, 0.6690528988838196, 0.39699453115463257, -0.295089453458786, -0.16769787669181824, -0.28230905532836914, 0.6226038932800293, 0.8616473078727722], [-1.0692905187606812, 1.0893125534057617, 0.20197808742523193, -0.5713686347007751, -0.7218409776687622, -0.1664370447397232, -0.43744558095932007, -0.20743009448051453], [0.127966046333313, -0.6902344822883606, -0.27221688628196716, 0.9405724406242371, -0.9173130989074707, 0.5391865372657776, -1.247575044631958, -1.077846646308899], [-0.005203901790082455, 0.34285247325897217, 1.7434864044189453, 0.5334979295730591, 0.4776477813720703, 1.386149525642395, -0.2674510180950165, 1.147807240486145], [-0.5679663419723511, 1.1786848306655884, 1.5419232845306396, -0.18613868951797485, 1.3225743770599365, 1.1066535711288452, -0.9158740043640137, 0.38017967343330383], [-0.5104151964187622, 1.4117846488952637, 2.1652588844299316, 0.7508301734924316, -0.8234401941299438, 0.608127772808075, -1.1323096752166748, -0.21140830218791962], [0.5745202302932739, 0.8391517996788025, -0.5330402255058289, -0.8780681490898132, 0.12349684536457062, 0.3203687071800232, -1.1932122707366943, 0.6320854425430298], [-0.6290944218635559, -0.33093729615211487, 0.44836869835853577, 1.1893796920776367, -0.6976255774497986, -2.146514415740967, 1.1611886024475098, -2.2768921852111816], [0.9347610473632812, -0.011106176301836967, 0.4779239594936371, -0.33538755774497986, 1.9620347023010254, 2.285916805267334, 0.8905056118965149, 2.6433351039886475], [1.0223119258880615, 2.5713493824005127, 0.34323740005493164, 0.07853814214468002, 0.9205058217048645, 0.3103882074356079, -0.5093503594398499, -0.2127789407968521], [0.4627962112426758, -0.27298423647880554, -1.1098518371582031, -2.4940712451934814, -0.40426766872406006, 0.019428325816988945, -1.5258729457855225, -0.45156583189964294], [-0.44478315114974976, -0.6142958998680115, -0.34805309772491455, -1.8537030220031738, 0.46561092138290405, 0.4484243392944336, 2.8421621322631836, -0.20608237385749817], [0.5738343000411987, 0.35120201110839844, 0.19046920537948608, 0.14453016221523285, 0.3563102185726166, -0.5124994516372681, -1.6389272212982178, -0.4052180051803589], [0.014407186768949032, -0.4227001965045929, -0.15023325383663177, -0.8172345757484436, -0.9002220630645752, -0.4789639115333557, 0.0226602740585804, -0.3830203711986542], [-0.9058100581169128, 1.0814036130905151, 0.06429638713598251, 0.006550712510943413, -0.7132749557495117, -0.7607928514480591, 0.9298311471939087, 0.0943666398525238], [0.8215736746788025, -2.3039283752441406, -0.40513497591018677, 0.9448987245559692, 0.21605485677719116, -0.7402067184448242, -0.2434270828962326, -0.044134583324193954], [0.2005649358034134, 0.2589613199234009, -0.8996691107749939, 0.2846692204475403, 0.3008158206939697, 0.9488381147384644, 0.29108503460884094, -1.4642879962921143], [1.4348828792572021, -0.9381688237190247, -0.6443460583686829, -0.6525417566299438, 0.6677617430686951, 1.1440858840942383, -0.03949694707989693, -0.5593855381011963], [0.34907621145248413, -0.7498024106025696, 0.7507708668708801, 0.4503003656864166, 0.6429666876792908, 0.6284415125846863, -0.2903331518173218, 1.0075654983520508], [0.996036946773529, 0.8897716999053955, -0.025820352137088776, -0.5304960608482361, -0.143401637673378, -0.1327877640724182, -2.049820899963379, 1.0958032608032227], [0.1359494924545288, 0.03624375909566879, -0.2701130211353302, -0.17305153608322144, 0.8576912879943848, -0.6259546279907227, 0.24852660298347473, 0.03207516297698021], [0.1888962984085083, 0.7517452836036682, 0.010536223649978638, -0.7119218111038208, 1.055364966392517, 2.219409227371216, -1.1816822290420532, -1.82333242893219], [0.4810689687728882, 0.4260157346725464, -0.4979804456233978, 1.0554490089416504, 0.3972700536251068, -1.629647135734558, -2.1823196411132812, 0.01888165809214115], [1.4006543159484863, -1.2008695602416992, 1.709304690361023, 0.3995072841644287, 0.15071401000022888, 0.7275082468986511, -1.3654884099960327, -0.8102589249610901], [-0.728213906288147, -1.7333353757858276, -0.8023285865783691, -2.101116418838501, 1.0229629278182983, -0.4143050014972687, -0.18936768174171448, 0.2116815447807312], [-0.5467135310173035, -0.5291318297386169, 1.138063669204712, 0.9556791186332703, -1.4086860418319702, 0.8984512686729431, 1.3270448446273804, -1.177112340927124], [0.6284141540527344, -1.0957694053649902, 0.194500133395195, -0.3800427317619324, 0.592903733253479, 1.155776023864746, -0.5698330402374268, 0.8047451972961426], [-0.0050069247372448444, 0.7130500674247742, 0.3929824233055115, -0.3175608515739441, -0.5587764978408813, 0.23447583615779877, 1.7177766561508179, -0.41395965218544006], [0.805882453918457, 1.9281799793243408, 1.260275959968567, -0.5886760950088501, -1.2337782382965088, -0.8067518472671509, -0.9649004340171814, -0.37735220789909363], [0.2218172252178192, 1.3559492826461792, 0.8610746264457703, -0.363662451505661, 0.039764050394296646, 0.46878179907798767, -0.43844878673553467, 1.2614624500274658], [0.11270667612552643, 1.4053289890289307, -1.6590057611465454, 0.5899112224578857, -0.019318656995892525, 0.270289808511734, -0.10150686651468277, 0.6175954937934875], [0.18096297979354858, -0.5374600887298584, -0.6908532977104187, -0.09835165739059448, 0.9204962849617004, 1.1905744075775146, 0.2573780417442322, -0.5712411403656006], [0.8516814112663269, 0.42651501297950745, 1.211159348487854, -0.5543016195297241, 0.8209282159805298, -0.8502349853515625, -0.578855574131012, 0.4071780741214752], [0.47359609603881836, 0.5528525710105896, 0.09986478835344315, 2.302030563354492, -0.12652479112148285, 0.6542834639549255, -0.28994452953338623, 0.18950627744197845], [1.7486441135406494, 0.17332154512405396, 0.08585350960493088, -1.7177271842956543, 0.37763622403144836, 0.4511561393737793, -0.4267564117908478, 0.3092830181121826], [-0.2683894634246826, -0.43211764097213745, 0.7030728459358215, 1.013808012008667, 2.877192974090576, -0.9525319337844849, -0.2694481909275055, -1.2717994451522827], [-0.38687530159950256, -0.6260366439819336, -0.49261677265167236, -0.27364328503608704, 0.2639189064502716, -1.230103850364685, -1.2862677574157715, 1.1561098098754883], [0.3854106664657593, 0.26505807042121887, 0.7841082215309143, -0.2309104949235916, -0.604409396648407, -1.4417996406555176, 0.9429270625114441, -1.0768373012542725], [-0.06504455953836441, -0.25894173979759216, -0.05547985062003136, -0.41578298807144165, 0.3841625452041626, 0.9170160889625549, -0.459037721157074, -1.232645034790039], [-0.7334514260292053, 0.32377997040748596, 1.267546534538269, -0.3947267532348633, 0.7822417616844177, 1.7897714376449585, -1.5197972059249878, 0.16337676346302032], [-0.10329718887805939, -2.3209023475646973, 0.9940934181213379, -0.11192820966243744, -0.18830017745494843, -0.030296094715595245, 0.7458913922309875, -0.09751464426517487], [-0.7446109056472778, -0.2979030907154083, 0.25480231642723083, 0.8344922065734863, 0.6963713765144348, 0.28422704339027405, -1.0360227823257446, -0.38531506061553955], [1.594527244567871, 0.8842783570289612, -0.3577042818069458, -2.2683725357055664, 1.1250702142715454, 0.4739472270011902, 0.6045615673065186, 0.5569500923156738], [0.3969925343990326, -2.0744879245758057, 0.2814372479915619, -1.0470607280731201, -0.2575887441635132, 1.2700804471969604, 1.0574982166290283, 0.14415138959884644], [1.3446778059005737, -1.779187560081482, 1.4180752038955688, -2.080489158630371, 0.2102409452199936, 1.4443120956420898, -0.19309377670288086, -0.8243380188941956], [-0.009898209944367409, 0.9900888800621033, -0.0018834039801731706, -1.5614784955978394, 0.3604365587234497, -0.36890095472335815, -1.2426233291625977, -0.08450658619403839], [-1.1297141313552856, -0.6131344437599182, -1.8377084732055664, 0.3656109571456909, 0.759864330291748, 0.8358413577079773, -0.5265738368034363, -1.5422435998916626], [1.0304275751113892, 2.568657636642456, -1.4250586032867432, 2.3126001358032227, -0.27822956442832947, 0.04820633307099342, 1.5674246549606323, -0.7239087224006653], [0.25331035256385803, 1.2749207019805908, 2.215325355529785, 2.0085291862487793, 0.735153317451477, 0.012455048970878124, -2.123840093612671, 0.9838046431541443], [0.4029994010925293, -0.04977430775761604, 2.3443570137023926, 1.1020097732543945, 0.48498642444610596, -0.7044079899787903, -0.38979408144950867, -0.5928568243980408], [0.673328697681427, -0.31218692660331726, -0.14940866827964783, -0.6288859248161316, 0.8069409132003784, 0.5711342692375183, 1.5454468727111816, 1.8046133518218994], [-0.7217859625816345, -0.6139814853668213, -0.7774226665496826, 1.5323866605758667, -0.9368578195571899, 0.33367252349853516, 0.3335208594799042, 0.03208035230636597], [0.5242831707000732, -0.07583148032426834, -0.59298175573349, 0.27048563957214355, -0.814710259437561, -1.425223708152771, -1.185275673866272, 0.535026490688324], [-2.0241215229034424, 1.3174653053283691, -1.1848080158233643, 1.147802472114563, -0.7416799068450928, -0.5256150364875793, 1.590671420097351, -0.8445256948471069], [-0.254082053899765, 1.9432560205459595, 0.2811664938926697, 1.13955557346344, -2.4439547061920166, 0.9986122250556946, 1.295306921005249, 0.9386895895004272], [-0.2619146704673767, -0.1057538241147995, -1.299279808998108, 0.3780844807624817, -0.9193588495254517, 0.6431930065155029, 0.791543185710907, 1.1319117546081543], [2.496412754058838, 1.3742334842681885, -1.6024799346923828, -0.5766046047210693, 1.1279603242874146, -1.1413366794586182, -0.91112220287323, 0.36337777972221375], [1.2200263738632202, -0.16221188008785248, -0.4675110876560211, 1.59396231174469, -1.7481694221496582, -0.2661984860897064, 0.4547634422779083, -0.6876352429389954], [0.5175474286079407, -0.643427848815918, -0.07939784973859787, -1.146905541419983, -0.7251653671264648, -1.0731351375579834, -0.9809821248054504, -0.9558744430541992], [-0.02252657525241375, 1.5170069932937622, 0.334929883480072, -0.9686455130577087, 1.0099828243255615, -0.07164006680250168, -1.318027138710022, 0.6548104286193848], [0.8310638070106506, 0.8355337381362915, 0.49574100971221924, 1.430618405342102, 0.10834864526987076, 1.5264089107513428, 0.14109265804290771, -0.5365949273109436], [1.170586109161377, -0.8942368626594543, 0.5457375049591064, -1.678851842880249, 0.29076170921325684, 0.7534993290901184, -0.16410405933856964, 1.8133692741394043], [0.7041539549827576, -1.6525771617889404, 0.8010486960411072, -1.5302835702896118, 1.1733487844467163, 0.02191188372671604, -1.8205974102020264, 1.6125235557556152], [-0.12684683501720428, 1.105507493019104, -0.7632038593292236, -1.658219575881958, 0.5175105929374695, -1.111720085144043, -0.4866922199726105, 0.13481566309928894], [0.7668150067329407, 1.0278841257095337, -1.9938896894454956, 1.4990932941436768, -0.001774731557816267, -0.2485569268465042, 2.2954928874969482, 0.8353431820869446], [1.2969136238098145, 0.3891496956348419, 1.1160064935684204, 0.1413397490978241, 1.1487131118774414, 0.16988740861415863, -1.2487133741378784, -0.42879724502563477], [-1.2217775583267212, -1.2076261043548584, -1.267520785331726, 0.6024333238601685, 1.2986812591552734, 0.5748283863067627, 1.3753069639205933, 0.4334632158279419], [-2.014683485031128, -0.4719979465007782, 0.12393214553594589, -0.6124032139778137, 0.9538440704345703, -0.10219930112361908, 0.5932435393333435, -1.2578028440475464], [1.0883854627609253, -0.5196503400802612, -0.20923294126987457, -0.6064682006835938, -1.2072556018829346, 0.1633284091949463, -0.6420322060585022, -0.26878148317337036], [0.36757606267929077, -0.08412431180477142, 0.07845644652843475, 0.7245195508003235, 0.6833769083023071, -0.2641419768333435, -0.3902941048145294, 1.3760950565338135], [-0.2639613449573517, -0.5179153084754944, 0.7337671518325806, 1.8119006156921387, 0.021622734144330025, 1.7546106576919556, -0.6974920630455017, -0.7949033975601196], [0.7822118401527405, -0.22954599559307098, 0.14879247546195984, 0.19545193016529083, 0.5694425702095032, 0.5413908958435059, -0.5255693197250366, -0.06160372495651245], [-1.439771056175232, -0.9292914867401123, 1.2314282655715942, -0.44999849796295166, -1.1903917789459229, -1.3276569843292236, -0.45224231481552124, 1.237640619277954], [0.3683421015739441, 0.3612162470817566, -0.25277432799339294, 0.5768750905990601, -0.029559694230556488, -0.9778994917869568, -1.7629672288894653, 0.27540677785873413], [-0.9916161298751831, 0.8351172804832458, -1.4897661209106445, -0.2643427550792694, 0.21101757884025574, 1.6931493282318115, -1.0672746896743774, -0.5440661311149597], [-0.1414102464914322, -0.40676796436309814, -1.125488042831421, 1.9390853643417358, 0.71487957239151, 1.6461338996887207, 2.0245883464813232, 1.155296802520752], [1.047418475151062, 0.19760315120220184, -0.3195610046386719, -1.1165510416030884, -1.7315558195114136, 0.5133125185966492, 1.3448106050491333, -2.023674964904785], [1.6752222776412964, -1.248756766319275, -0.8949147462844849, -1.3549742698669434, 0.119442418217659, 0.8188827633857727, 0.03640449047088623, 1.6794114112854004], [0.5889316201210022, -0.6014199256896973, 1.2183866500854492, 0.9389408230781555, -0.3271113336086273, -0.6059066653251648, 0.5117213726043701, 0.42568618059158325], [-0.299375057220459, 0.5979200601577759, 1.6480889320373535, -0.48499760031700134, 0.8487957715988159, -0.18534301221370697, -0.22427284717559814, -0.6898714900016785], [-1.0417011976242065, 0.7760084867477417, -0.8062177300453186, 0.024410368874669075, 1.5005061626434326, -2.0463438034057617, -0.05785754323005676, 0.2639402151107788], [0.31462496519088745, -1.3204543590545654, 1.5753107070922852, -0.9965674877166748, 0.1234714537858963, -0.015844151377677917, 0.43479374051094055, -1.5147662162780762], [-1.4822410345077515, 0.25048214197158813, -0.06728994846343994, 0.8721770644187927, -0.4387785792350769, 2.2014849185943604, -0.6914160847663879, 0.744903564453125], [1.6909540891647339, -0.3994814455509186, 0.8299999237060547, 0.3812473714351654, -0.3857824206352234, -1.0216628313064575, -0.11340843141078949, 1.1869317293167114], [0.9611743092536926, 0.48308324813842773, 0.5406250357627869, -0.7076928019523621, 0.3212260603904724, -0.557009220123291, 0.7158411145210266, -0.5525191426277161], [-0.7269703149795532, 0.8254331946372986, 0.8539828658103943, -0.6593995094299316, 0.35366013646125793, 0.16822928190231323, -0.5963497161865234, 1.4703655242919922], [1.521022081375122, -0.5286790132522583, -0.9298908710479736, -1.4546630382537842, -0.6309178471565247, -1.0214835405349731, 0.926744282245636, -0.5006264448165894], [-2.0308289527893066, 0.5337246060371399, 1.2118300199508667, 0.4024132490158081, 1.0763640403747559, 1.4807026386260986, 0.6584159731864929, 0.7792165875434875], [1.6361048221588135, -0.5966443419456482, -0.8504359126091003, -0.1265985667705536, -1.6873568296432495, 0.07915201038122177, 0.11570890992879868, 0.30891984701156616], [0.029756292700767517, -0.9471812844276428, 0.6787979006767273, 0.8235626220703125, -1.5570141077041626, 0.6921533942222595, -0.7921214699745178, 0.09484193474054337], [1.1811834573745728, 1.6750212907791138, 1.6286251544952393, -0.8876007199287415, -0.3042196035385132, -0.5357558727264404, -1.503868579864502, -0.18852809071540833], [-0.885807991027832, -0.9538531303405762, -0.18504202365875244, -2.004119873046875, -0.2906545102596283, -0.6743436455726624, -2.1684858798980713, -0.4777017831802368], [0.7057723999023438, 2.281372308731079, 1.3570090532302856, -0.22454623878002167, -1.0333664417266846, 1.2121307849884033, 0.6605041027069092, 0.6316668391227722], [0.2900877594947815, 0.08744348585605621, -0.696100115776062, 2.456313371658325, 2.530945062637329, 1.4527968168258667, -0.2204757034778595, 0.6164919137954712], [-2.069735050201416, -0.5461077094078064, -0.9702299237251282, -0.06861782819032669, -0.27060338854789734, -0.018806176260113716, 0.3852996826171875, -0.11034756153821945], [0.4781181216239929, 0.42367053031921387, -0.5275819897651672, 0.16658338904380798, -1.5026599168777466, 1.4806244373321533, -0.13744454085826874, -1.6878323554992676], [-0.3328966498374939, -0.2499236762523651, 0.9331438541412354, -1.448567271232605, 0.026790104806423187, 0.6443144679069519, -0.6760187745094299, -0.0269540473818779], [0.22541698813438416, -0.08922216296195984, -0.8805606365203857, -0.4232384264469147, -0.15784426033496857, -1.2034339904785156, -0.9578679800033569, 1.6728196144104004], [1.2206032276153564, -0.00926921982318163, -0.032114092260599136, -0.9843196272850037, 1.1473441123962402, 0.6268665194511414, -0.8882394433021545, 1.66580069065094], [0.8314234018325806, 0.7107171416282654, 0.43391767144203186, -0.1809297353029251, -0.18478363752365112, -0.9546067714691162, 1.8776806592941284, -0.6737693548202515], [1.5780256986618042, 0.27058473229408264, 1.0856401920318604, 1.0989574193954468, 0.5986669659614563, 1.543355107307434, 0.7805989980697632, 1.018467664718628], [0.43476611375808716, 0.3274425268173218, -0.06305036693811417, -0.2152106910943985, 0.6134507060050964, -0.9386391639709473, 0.6401903629302979, 1.4182748794555664], [-0.9973816871643066, -1.7783434391021729, 0.26176804304122925, 0.9028847813606262, -0.461174339056015, 1.4220672845840454, -0.47654783725738525, -1.5395525693893433], [0.1456986963748932, 0.4490715563297272, -1.1357396841049194, -1.9149247407913208, -1.041062831878662, 1.3894193172454834, 1.6801356077194214, 0.8174170851707458], [0.2364167720079422, 1.701256513595581, -0.7894731760025024, -1.5581550598144531, 0.036806534975767136, 0.5231810212135315, 0.22485260665416718, 0.2580476701259613], [1.8422318696975708, -0.021351944655179977, 1.0457628965377808, 0.38007065653800964, 0.8494729995727539, -0.12693500518798828, -0.5473929643630981, 0.5031039118766785], [-0.05201995000243187, 0.3149849474430084, -0.7338293194770813, 0.9666099548339844, 0.0436411090195179, 0.258513480424881, -1.1787705421447754, -0.6277408003807068], [2.0117502212524414, 2.125406503677368, -1.0516865253448486, 0.1603352278470993, -0.7556295394897461, 0.7239117622375488, 0.9876301288604736, 0.5675589442253113], [0.6502144932746887, 1.1316592693328857, 1.4851783514022827, 0.734630286693573, 0.1362139880657196, -1.5831880569458008, 1.370811939239502, -0.3780301511287689], [0.6526215076446533, 0.3584701716899872, 0.4735572338104248, 0.6234757900238037, -0.7340546250343323, 0.6835001111030579, 0.33102700114250183, -0.36919111013412476], [0.3759864270687103, -0.3323141932487488, -0.04825691878795624, -0.6873050332069397, -0.7063486576080322, -1.28868567943573, 0.9444831013679504, 0.12684085965156555], [1.1265000104904175, -0.17986081540584564, 0.3141421973705292, 2.355184555053711, 0.4383777379989624, -1.0843955278396606, 0.6272676587104797, 0.1325107365846634], [-1.2094523906707764, -1.9275081157684326, 1.1582192182540894, 1.5750967264175415, -1.3394414186477661, 1.4814121723175049, 0.4972924292087555, 1.5560829639434814], [-3.0810904502868652, 2.584043264389038, 1.9099633693695068, 0.08882803469896317, 0.4154588282108307, 0.6429044604301453, 1.3784734010696411, 0.8302733898162842], [0.17562097311019897, 0.2603227496147156, 0.17107000946998596, -0.01315249502658844, 1.008156418800354, 1.2048826217651367, -0.7471878528594971, -0.855773389339447], [2.698791980743408, 0.04147918149828911, -0.5761659145355225, -2.0478363037109375, 1.4107667207717896, -0.8336660265922546, -1.6311033964157104, -0.82956862449646], [-0.04856749251484871, -1.3272813558578491, -0.5380903482437134, 1.9179514646530151, -0.6469959020614624, 0.3259985148906708, -1.030310869216919, 1.8328033685684204], [0.12437961995601654, -0.1889984905719757, -1.4661462306976318, 0.9306204915046692, 0.8103839159011841, 0.4901496469974518, 1.6571489572525024, 1.063673973083496], [-0.46205615997314453, -0.7718790173530579, 0.3390766680240631, 0.019122451543807983, 0.319227010011673, -1.025421142578125, 0.13015660643577576, 0.6105398535728455], [-0.236162468791008, -0.34691599011421204, 1.4626909494400024, -1.7314386367797852, -0.4904429614543915, 0.08473774045705795, 1.0125572681427002, -0.07278299331665039], [-0.24384500086307526, -0.04961176961660385, -0.34391286969184875, 0.8772834539413452, 0.2629087269306183, 0.06382538378238678, 1.4950441122055054, 1.0980304479599], [0.25676229596138, -1.752077579498291, -3.0006306171417236, 0.09513666480779648, -0.1187305897474289, 2.4243123531341553, -0.2815774977207184, -0.8904399275779724], [-0.31414222717285156, -0.32330095767974854, -1.4638280868530273, 1.8250535726547241, 0.1949865072965622, 0.7353823184967041, -0.44383329153060913, 1.6847108602523804], [-1.0410144329071045, 0.7812073826789856, 0.8949547410011292, -0.4840323030948639, -0.722511887550354, -0.20997591316699982, 0.4719107449054718, 0.1565978229045868], [0.656548798084259, 0.9518646597862244, 1.3793492317199707, 0.762525737285614, -0.5663597583770752, 0.009290437214076519, 0.2508786916732788, 2.223356008529663], [-1.008740782737732, 1.7270479202270508, -0.4364997148513794, 1.3041318655014038, -1.9815373420715332, 0.31679970026016235, -1.6431266069412231, 1.2353979349136353], [0.18517208099365234, 1.13252592086792, 0.7135061025619507, -2.200751781463623, -0.4586229622364044, 0.20866671204566956, -0.8782302141189575, 0.7487035989761353], [-0.3010333776473999, -1.2588353157043457, 0.7612356543540955, -2.2138590812683105, -0.43999505043029785, -0.9264640212059021, 1.8644671440124512, -1.1378653049468994], [-0.26794207096099854, -0.37610751390457153, 1.6948400735855103, 0.4707362651824951, -0.14171595871448517, 0.7672625780105591, -1.3007490634918213, -0.15895149111747742], [0.11818026006221771, 0.19614426791667938, -0.5068166851997375, 0.9582255482673645, -0.42914852499961853, 0.9387273192405701, -1.788565754890442, 0.4511553645133972], [-1.272091269493103, 1.5029205083847046, -1.3679405450820923, 0.30029022693634033, 0.47448599338531494, 0.6149166226387024, 1.162666916847229, 0.7452923059463501], [-0.029844924807548523, 0.38410890102386475, 0.4545024633407593, -0.44128161668777466, -0.7385048270225525, 0.6953263878822327, 1.68578040599823, 1.691052794456482], [0.2053641825914383, -0.9770359992980957, 0.5269737243652344, 0.8258283734321594, 1.1606240272521973, -1.062540888786316, -1.6430809497833252, 1.9000749588012695], [-0.8708942532539368, -0.10842897742986679, -0.10460027307271957, 1.7309175729751587, -1.017375111579895, -0.7745881080627441, -0.5952349305152893, 2.770886182785034], [1.4952210187911987, -1.5763458013534546, -1.4476871490478516, 0.16212819516658783, 2.2152602672576904, -0.7081558108329773, 1.4307724237442017, 0.5249650478363037], [0.09681448340415955, 0.711682140827179, -0.5123084783554077, -0.44600415229797363, 0.62109375, 0.849524974822998, 0.8516800403594971, 0.3577534556388855], [0.8325110077857971, -0.6514227986335754, -0.23028399050235748, -0.4160500466823578, 0.05133017525076866, 0.2738202214241028, -0.08869641274213791, 0.22809992730617523], [-3.075444459915161, 0.11406145244836807, 0.5939792990684509, 0.38583454489707947, 0.1527300924062729, 1.1931788921356201, -0.029704058542847633, -1.6720941066741943], [0.9788303375244141, -0.5406467914581299, -1.1626392602920532, -0.2835720479488373, 0.12144435942173004, -0.8622706532478333, -1.0461350679397583, -0.8657816648483276], [1.6531857252120972, 0.3025226294994354, 0.3451492190361023, -0.3064805865287781, -0.012761968187987804, 0.5270192623138428, -0.5596306920051575, 0.5809727907180786], [1.526734471321106, 0.2184862643480301, 0.8469110131263733, 0.10607681423425674, -0.2328099012374878, 0.46404367685317993, -1.275068998336792, 0.6714633703231812], [2.374025821685791, -0.5303343534469604, 1.0873509645462036, 0.21064874529838562, 0.7417325973510742, 0.08059047162532806, 0.23189115524291992, 0.7212160229682922], [0.4966756999492645, 0.39135804772377014, -0.31390270590782166, 1.7883399724960327, -0.692646861076355, 1.1416250467300415, -0.7797420620918274, 1.081850528717041], [-0.6646647453308105, -0.03517685458064079, 1.899821162223816, 0.39531177282333374, -1.0178605318069458, 0.3236209750175476, -0.2730101943016052, 0.3402164578437805], [0.8375414609909058, 0.6794258952140808, 0.6438579559326172, 0.9593054056167603, 0.35343390703201294, -1.1506954431533813, 1.2555418014526367, 1.2560850381851196], [0.4273683428764343, 0.5691816210746765, 1.0724612474441528, 0.07974656671285629, 0.26580333709716797, -0.05536632612347603, -0.2727425992488861, -0.7717739343643188], [-0.23756442964076996, 0.1771257072687149, -0.7017691731452942, -0.1362573355436325, -1.8170794248580933, -1.137247920036316, -0.25812238454818726, -1.5731539726257324], [-0.6732553839683533, -1.1904373168945312, 1.3575043678283691, 1.9431763887405396, -0.8779888153076172, -1.012035846710205, -0.695172131061554, -0.3040485084056854], [-0.14398254454135895, 1.401990294456482, -0.651146650314331, -1.0505363941192627, 1.605212688446045, -0.348583847284317, 1.1530042886734009, -1.0985243320465088], [-0.3627275228500366, 1.3165417909622192, 0.5404231548309326, 2.0582103729248047, 1.667602300643921, 2.448016881942749, -0.697242021560669, 2.0769131183624268], [-0.6131758093833923, 0.3302103877067566, -0.11545191705226898, 1.029006838798523, 0.8796459436416626, -0.8408886790275574, -0.2358844131231308, 0.3361028730869293], [0.18745020031929016, -0.2547977864742279, -0.25542888045310974, -0.9664531946182251, -0.5276089906692505, 0.8781973123550415, -0.22350692749023438, 0.16002491116523743], [0.032798465341329575, 0.6200730204582214, 0.39049869775772095, -0.09028401970863342, -0.5036953091621399, -0.47215452790260315, 1.316108226776123, -1.1451975107192993], [0.8275641798973083, 0.33834537863731384, 0.39547520875930786, 2.018242835998535, 1.5691150426864624, 1.6237308979034424, 1.8851743936538696, -0.3156762719154358], [-1.022135615348816, 1.3599625825881958, -0.5193729996681213, -0.8295860886573792, -1.8469160795211792, 0.5189220309257507, 1.434939980506897, 0.32766103744506836], [0.4204927682876587, -0.23840032517910004, 1.252540946006775, 0.052163247019052505, 0.1080465018749237, 0.12465247511863708, -0.3885294795036316, 0.3608885407447815], [-0.7708569169044495, -1.0680264234542847, 0.01906444877386093, 0.4504009187221527, 2.479705333709717, 0.6739874482154846, 0.37236884236335754, -0.9392135739326477], [-1.5691567659378052, 0.7249312996864319, 0.9584572911262512, -0.21550890803337097, -0.6710473299026489, 1.0054641962051392, 0.9738855361938477, 1.2032122611999512], [1.4830948114395142, 0.5056135654449463, -0.1789853274822235, 0.5439825654029846, -0.2156120389699936, -1.4645872116088867, -0.45856064558029175, -0.8399760127067566], [1.0893179178237915, 0.24197480082511902, -0.5546174645423889, 0.8922102451324463, -0.5814225673675537, 0.31820932030677795, -1.464890956878662, -0.5652115345001221], [0.7543608546257019, -0.5843625664710999, 0.31888219714164734, -0.5496626496315002, -0.6186855435371399, 0.09112212806940079, 2.3680851459503174, 0.8286955952644348], [-0.2948518693447113, 0.08884385973215103, 0.9962268471717834, -0.4923454523086548, -0.08799519389867783, -1.236746907234192, -1.8493120670318604, 0.3319043517112732], [0.9479086995124817, -0.08856549859046936, -0.28333330154418945, 0.1555456668138504, -0.5158466696739197, -0.6804214715957642, 0.11011018604040146, 1.5371720790863037], [0.23896996676921844, 1.5448423624038696, -0.44925183057785034, 0.7766723036766052, -0.7970913052558899, 0.560692548751831, 0.7900442481040955, 1.0569984912872314], [0.2292865365743637, 0.707859218120575, 0.1413046419620514, 1.3683819770812988, -0.8805944323539734, 0.824860692024231, -1.1134263277053833, -0.35440683364868164], [0.797663688659668, -1.5612375736236572, -0.13808389008045197, -0.7791944146156311, 0.3134097158908844, -0.2759634554386139, -0.4480862617492676, -0.13460491597652435], [-1.1810156106948853, -1.234075903892517, 0.2566705048084259, -1.1854147911071777, 1.4513416290283203, 0.2515130341053009, 1.1111036539077759, -1.0913394689559937], [0.1511046141386032, 0.007685324177145958, 0.1340264528989792, -0.377841055393219, 0.28774532675743103, -0.17069223523139954, 0.49756544828414917, -0.5442430377006531], [0.4783836305141449, 0.5341041088104248, 2.0805485248565674, -1.1029086112976074, -0.13821856677532196, 1.4415984153747559, -0.4832358956336975, 0.08280142396688461], [0.9120625853538513, -1.4524874687194824, -2.1967010498046875, -0.23945067822933197, -0.6056600213050842, -1.905123233795166, -0.14768001437187195, 0.014280805364251137], [0.12511374056339264, 1.7586994171142578, -0.5303347110748291, -2.3084936141967773, -1.8697997331619263, -0.2683122158050537, 0.16064366698265076, 0.5990349650382996], [1.3046444654464722, -1.4441500902175903, -0.13983389735221863, -0.21373245120048523, 1.5956887006759644, -1.2251619100570679, -0.5535997152328491, -0.38269054889678955], [1.7106702327728271, 0.5730488896369934, 1.1550867557525635, 0.2067250907421112, 0.9939852356910706, -0.6260943412780762, 0.5123827457427979, -0.8280360102653503], [0.9191922545433044, -1.1565077304840088, 0.3857021927833557, -0.23615410923957825, 1.7263039350509644, -1.4426019191741943, 1.153434157371521, -0.8025432825088501], [-1.5459003448486328, -0.5654227137565613, -1.0683398246765137, 0.030912885442376137, -0.4287724792957306, 1.3086990118026733, 1.1444119215011597, 0.1952926069498062], [0.32299113273620605, 1.3371554613113403, 1.6731343269348145, 1.5464307069778442, -0.2640671133995056, -0.9052299857139587, -1.02507483959198, 0.9911677837371826], [-0.003668292658403516, -2.6849799156188965, -1.1154643297195435, 1.0407638549804688, 0.7626004219055176, 0.19440573453903198, -1.9751545190811157, 0.5084770917892456], [0.819830060005188, -1.8169152736663818, 0.45484259724617004, -0.4650592803955078, -1.2896779775619507, -0.04986754432320595, 0.4209609627723694, -0.42722487449645996], [0.8513679504394531, 0.19137687981128693, 0.7327555418014526, -0.8479138612747192, 0.1811426430940628, -2.076977491378784, 0.09047539532184601, -1.0571315288543701], [0.157410129904747, -0.9831701517105103, -0.6506217122077942, -0.022123724222183228, -0.0665469691157341, 1.0472321510314941, -1.7904019355773926, 1.8097513914108276], [-1.2110620737075806, -0.8090900182723999, -0.3636154234409332, -1.8346437215805054, -0.42388880252838135, 1.4702948331832886, -2.100492477416992, 0.1643359363079071], [-0.017850792035460472, -1.3905164003372192, 0.06652773171663284, 0.4161324203014374, 0.7701910734176636, -1.1962023973464966, 1.1726970672607422, -0.6636601090431213], [0.18912425637245178, -0.491561621427536, -0.1803637146949768, -0.6614829301834106, 0.4329913556575775, -0.13759444653987885, -0.37636205554008484, -1.0546990633010864], [1.297922968864441, -1.9344736337661743, 1.654112458229065, 1.4862298965454102, 0.037287767976522446, 0.7757458090782166, -0.8428913354873657, -1.2807525396347046], [-1.0205857753753662, 0.3404412567615509, -1.1157574653625488, 2.5377094745635986, -0.917437732219696, 0.33467206358909607, 0.9547212719917297, -0.8529514670372009], [-0.46146759390830994, 0.019060641527175903, -0.9000529646873474, -0.7978154420852661, 0.12695030868053436, -0.4416438639163971, -0.3397582769393921, 0.8734787702560425], [-1.9663530588150024, 1.5686739683151245, 0.17099575698375702, 0.0006124629871919751, -0.03950214758515358, 0.8121763467788696, 2.466104507446289, 1.903739333152771], [0.5369753241539001, 1.7278128862380981, -1.3497514724731445, 0.31777554750442505, -1.5451579093933105, -0.8085846304893494, -0.1694760024547577, -1.422273874282837], [1.078979253768921, -0.271732896566391, 0.6839122176170349, 0.417309045791626, 1.2484517097473145, -0.8935521841049194, 0.8754113912582397, 0.034538693726062775], [-0.19058658182621002, 0.5009121894836426, -0.4978564381599426, 2.144530773162842, 0.13332562148571014, -1.97786283493042, -0.466497004032135, -0.31810542941093445], [-1.065845012664795, 0.4860880970954895, 0.7281827926635742, 0.4587048888206482, -0.5520063042640686, 0.12801425158977509, 0.0022169228177517653, 2.036302328109741], [-1.8898570537567139, -2.0046262741088867, -0.012511125765740871, 0.890437126159668, -0.9998286962509155, 0.20260852575302124, -0.013025506399571896, 0.4612785279750824], [-1.507459044456482, 0.668430745601654, -0.25425100326538086, -1.1672996282577515, 2.302445650100708, -0.9142683744430542, 1.3481221199035645, -1.136684775352478], [0.22073791921138763, -1.5847712755203247, 0.005469832569360733, 0.652993381023407, 0.9127197265625, 0.5498961210250854, -1.719600796699524, -0.5269829630851746], [-0.14709848165512085, 0.9959413409233093, -1.2420669794082642, 0.30269280076026917, 0.5572394132614136, 0.7242938280105591, 0.5322431325912476, -0.5840322971343994], [0.6445144414901733, 0.959792971611023, -0.4184030294418335, -1.7794177532196045, -1.0521496534347534, 0.5827885866165161, -0.706713080406189, 0.9484006762504578], [-0.24563723802566528, -0.4053240418434143, 0.9686067700386047, 0.7174842357635498, 0.5623261332511902, 0.9318108558654785, -0.9501004219055176, 0.07473434507846832], [-0.05266176164150238, -0.422720730304718, -0.24866066873073578, -0.6290907263755798, -0.0023605898022651672, -0.5059103965759277, 0.9844160676002502, 0.5301871299743652], [1.457819938659668, 1.469934105873108, 1.1056914329528809, -1.0547726154327393, -0.3334733545780182, 0.2027832567691803, -1.5520275831222534, 0.14774483442306519], [0.08943655341863632, 0.15260480344295502, 1.4917956590652466, -0.48397383093833923, -0.01364942081272602, -0.44405943155288696, -0.27506116032600403, -0.761528491973877], [-1.033661961555481, 1.302988052368164, -1.6228182315826416, 0.13424867391586304, 0.3537742495536804, 0.6669729948043823, -1.427879810333252, 0.5124143958091736], [0.2942076623439789, -1.097038745880127, -0.113441601395607, -1.2431432008743286, 0.3605928122997284, -0.2811623215675354, -0.43765828013420105, 0.6247261166572571], [-1.05177903175354, 3.9071269035339355, 1.1289397478103638, 0.463570773601532, 0.6467713117599487, 0.12116852402687073, -2.5568742752075195, 0.4142494797706604], [-0.7941503524780273, 0.48755592107772827, -1.6051288843154907, 1.055243968963623, 0.9662265777587891, 1.2598408460617065, -0.25158512592315674, -2.455848217010498], [-0.5899257659912109, -0.02095806412398815, 0.12969906628131866, -1.74277663230896, -0.327538400888443, 1.3558080196380615, -0.6581282615661621, -0.3223303258419037], [-0.8404099345207214, 1.040586233139038, -0.1696990728378296, 0.6686056852340698, -0.8060554265975952, 0.21477892994880676, 0.30704209208488464, 0.5642387270927429], [-0.39448612928390503, 0.8520565629005432, 0.47451817989349365, -1.3204929828643799, 0.14215700328350067, -0.03621334210038185, 0.7477242350578308, -0.292650043964386], [-1.5858817100524902, -0.547629177570343, -0.047283582389354706, 0.10074649751186371, -0.45314353704452515, -0.9712076783180237, 1.2909640073776245, -1.0736610889434814], [-0.5112877488136292, 0.39558184146881104, -1.2795857191085815, -0.9706450700759888, -1.4959501028060913, 1.0740493535995483, -0.16720551252365112, -0.5806517601013184], [-1.0376298427581787, 0.5732991695404053, 1.3395061492919922, 0.7133283019065857, -0.23227761685848236, 1.1718343496322632, -1.1368948221206665, -2.043534755706787], [-1.298667550086975, 0.5580444931983948, 1.4716277122497559, 1.130537986755371, -1.0794737339019775, -0.9142538905143738, 0.6490485072135925, -1.1066786050796509], [-1.206666111946106, 0.8931307196617126, -2.961843967437744, 0.6558861136436462, -0.37205958366394043, -0.22532841563224792, 0.38155385851860046, 0.7080431580543518], [-1.3216798305511475, -0.5679197311401367, 0.778211772441864, 0.9082036018371582, -0.06552417576313019, -0.19094395637512207, -0.01250722911208868, -1.1758157014846802], [0.9904309511184692, 0.5606228709220886, -1.1328202486038208, 0.9044535160064697, -0.16139866411685944, 0.695773184299469, -1.2880617380142212, -0.015522447414696217], [-1.0589762926101685, -1.4629771709442139, 0.4068135619163513, 0.40597110986709595, -0.717807948589325, -0.04503412917256355, 1.2157379388809204, -1.067949652671814], [0.520736575126648, 0.5719025731086731, 0.4328552186489105, -0.8741248250007629, 1.7912105321884155, 0.6075193285942078, 0.2879190742969513, -0.28287097811698914], [-0.6640811562538147, -1.0840415954589844, -1.4510055780410767, -0.6099615693092346, 1.1642390489578247, 0.6080286502838135, -1.2763755321502686, -0.815483570098877], [0.012937751598656178, -1.3997114896774292, 0.476039856672287, -0.7663651704788208, -1.131555199623108, 0.37584567070007324, -0.38665181398391724, -0.7797247171401978], [1.3165054321289062, 0.7150105834007263, -0.12572605907917023, 1.8377037048339844, 0.5172181725502014, -0.8932492136955261, 0.9141892790794373, -1.1249098777770996], [0.34888583421707153, -1.1473352909088135, -0.5339008569717407, 0.08098861575126648, -0.5418114066123962, 1.194048285484314, -0.6827269196510315, 0.5640448927879333], [-1.1043683290481567, -1.1109519004821777, -1.4444977045059204, 0.7654449343681335, -1.1205848455429077, 0.03609835356473923, -1.1863361597061157, -0.3723691403865814], [2.5069212913513184, -0.9014427065849304, -0.010733517818152905, -0.7491488456726074, -1.673244833946228, -0.058722421526908875, 1.3084776401519775, 0.5791525840759277], [-1.1963688135147095, 0.5123059749603271, -0.018296372145414352, -0.9009177088737488, -0.7071473598480225, 0.3344017267227173, -0.9285868406295776, 0.604314386844635], [-0.7556869983673096, -0.7638963460922241, -1.8139585256576538, -0.20729240775108337, 0.1713610291481018, -1.1101211309432983, 0.35499224066734314, 2.1130404472351074], [0.5457817316055298, 0.29807862639427185, -0.2989630401134491, -1.1633163690567017, 1.6385232210159302, 1.0447691679000854, -0.3282286822795868, -0.24362806975841522], [0.7392802238464355, -0.7260721921920776, 0.2652139663696289, -0.36765700578689575, -0.38529473543167114, -0.7274574041366577, -0.10313745588064194, 0.755632221698761], [-0.565645694732666, 0.6710500717163086, 1.4561009407043457, 0.04779337719082832, 0.28591564297676086, -0.9362730383872986, -0.9605563879013062, 1.2764160633087158], [0.8178997039794922, -1.70195472240448, 0.4238298833370209, 0.547015368938446, 1.8003536462783813, 1.882920503616333, 0.6023762226104736, 0.04021003469824791], [0.11519969254732132, -0.1858210563659668, -1.3174118995666504, -0.9364780187606812, 0.6905913949012756, 1.0259878635406494, -0.4498218894004822, -0.020290249958634377], [-0.19217082858085632, 0.7806271314620972, -1.2964636087417603, -1.1143440008163452, 0.39930564165115356, -0.6802836656570435, 0.5852633714675903, -0.6662877798080444], [-2.2243106365203857, -0.5581682920455933, 0.3313143253326416, -0.8807323575019836, 0.3176337778568268, 0.4291019141674042, 0.6256955862045288, -0.5126813650131226], [-0.6925081014633179, -0.9402305483818054, 1.356242060661316, -0.5209975838661194, 0.6978026628494263, -0.7404745817184448, 0.5999962091445923, -0.6489931344985962], [0.30410951375961304, 0.41508346796035767, 0.19001798331737518, 1.0298062562942505, -0.5011597275733948, -1.103550672531128, 0.9755099415779114, -0.7315713167190552], [-1.0167049169540405, -1.1150776147842407, 1.228690266609192, -0.1633719503879547, 0.7293694019317627, -1.109083890914917, -0.91447913646698, 0.4167501628398895], [-0.46667909622192383, 0.2397875189781189, -1.7838850021362305, 0.8981891870498657, 0.8595256209373474, 0.28291162848472595, -1.0104905366897583, -0.012897811830043793], [-0.6071439981460571, 0.6671633720397949, -0.3915247321128845, -1.2243075370788574, 1.0737322568893433, -0.7088629007339478, -0.39108291268348694, -0.7293868660926819], [0.035266585648059845, -0.3756044805049896, 0.7791587114334106, 0.5118021965026855, 0.16224871575832367, -1.5914027690887451, 1.0831509828567505, 1.0579992532730103], [-0.8856355547904968, 1.651246428489685, 0.5806810855865479, -0.8297207951545715, -1.0925592184066772, -0.6927865147590637, 0.3999442458152771, -0.8998295068740845], [-1.1093783378601074, -0.8987548351287842, 0.8302782773971558, 0.36715471744537354, 0.5664175748825073, 0.3632062077522278, 0.42489126324653625, 0.11341769248247147], [-1.0511281490325928, 1.136531114578247, 2.0524895191192627, 0.6990246772766113, 0.10582269728183746, 0.44839054346084595, -0.9445815682411194, 0.25349605083465576], [0.5241756439208984, 2.745778799057007, 0.9877811074256897, 0.7162836194038391, -1.357601523399353, 0.11221291124820709, 0.9106202125549316, -0.8257325291633606], [0.44835734367370605, 0.040292464196681976, -1.4027425050735474, -0.7437624931335449, -1.6457525491714478, -0.6885963082313538, -0.20196405053138733, 0.4891211986541748], [0.05728217959403992, -1.0832775831222534, -0.25881797075271606, 0.9480407238006592, 0.3404516875743866, -0.6867353916168213, -0.9800427556037903, -1.2478872537612915], [-0.5546261072158813, 0.2782139182090759, 0.2517441511154175, 0.18061643838882446, 0.5131638050079346, 1.046860933303833, 1.3639386892318726, -0.08927197754383087], [-0.1752062886953354, 0.20506782829761505, 0.4075930118560791, 1.2428128719329834, 0.3606448173522949, 0.3953884243965149, -0.6717031002044678, 0.5533689260482788], [-0.398336797952652, 0.044821131974458694, -0.762813150882721, 0.3597277104854584, 0.5536232590675354, -0.834118664264679, -1.3661608695983887, 1.1336947679519653], [-0.43964314460754395, -0.9109078049659729, -1.0954450368881226, -0.8483335375785828, 0.24686388671398163, -1.3798449039459229, 0.47163981199264526, -0.3179894685745239], [-0.14943112432956696, -1.0664387941360474, 0.9690598249435425, -0.24091239273548126, -1.3868948221206665, 1.2367513179779053, -0.5125247836112976, -1.0653003454208374], [0.9165935516357422, 0.15250511467456818, 0.7963941693305969, 0.38462382555007935, 0.5051264762878418, 0.11613061279058456, 0.5756656527519226, -0.31058552861213684], [1.1039283275604248, 0.8974552154541016, 0.7989572286605835, -0.705194354057312, 1.5927711725234985, -2.7810332775115967, -0.9503606557846069, -0.6811239719390869], [0.9406903982162476, -0.8859739303588867, -1.1439436674118042, -0.7869173884391785, 1.4744035005569458, -1.2627434730529785, 0.8431912660598755, -0.13038797676563263], [0.776045024394989, 1.3727893829345703, 0.708221435546875, 0.23788028955459595, -0.24592353403568268, 0.6058210134506226, -0.5706672072410583, -1.3908559083938599], [1.5197410583496094, -0.6439883708953857, -1.47742760181427, 0.14032770693302155, -1.9667186737060547, -1.8705817461013794, 1.5266684293746948, 0.08575177937746048], [0.9942030310630798, 1.1259520053863525, 0.475819855928421, 1.0885870456695557, 0.932101309299469, 0.025548258796334267, 0.7501230835914612, 0.8453670144081116], [0.36733901500701904, 0.7521599531173706, -0.5247064828872681, -0.30323174595832825, 0.8193495273590088, -0.3536914885044098, 0.9570022225379944, -1.757744312286377], [-0.09246435761451721, -1.3920741081237793, -0.055699173361063004, 0.31381163001060486, 0.3742940127849579, -0.8266443610191345, -0.3756233751773834, -0.8348450064659119], [1.0366768836975098, 0.051495034247636795, -0.8091335296630859, 2.34431529045105, 0.6889848709106445, 1.1585493087768555, 0.18644043803215027, -0.6001063585281372], [0.3426147699356079, 0.37845784425735474, 0.8415972590446472, 0.9173374176025391, 0.8051710724830627, 1.76382315158844, 1.718579888343811, -0.7006882429122925], [0.9580403566360474, 0.551308810710907, 0.9561623334884644, -0.1339206099510193, 0.2368113249540329, -0.2654437720775604, -0.4194931089878082, 1.1227649450302124], [-1.5808279514312744, 1.2609144449234009, -1.3400638103485107, 0.6416328549385071, 1.6057229042053223, 1.1092932224273682, -0.011202084831893444, -0.5223045349121094], [0.8844388723373413, -0.5431160926818848, -2.085341215133667, -0.5601313710212708, -0.09552536904811859, -0.8575252890586853, -1.7866215705871582, -2.4499599933624268], [0.19123248755931854, 0.019954293966293335, -0.1744663566350937, 0.34422698616981506, -1.2496528625488281, 0.09850858151912689, 1.7092267274856567, 0.5083096623420715], [-0.23653291165828705, -0.3052033483982086, 0.1519748568534851, -0.585347056388855, -1.1586647033691406, -0.6355292797088623, 0.31930482387542725, 0.5536285042762756], [0.16556596755981445, 0.4234961271286011, 1.4418110847473145, -1.1869094371795654, 1.0139127969741821, -0.20752693712711334, -1.2101720571517944, 0.19853900372982025], [-1.4126759767532349, -1.5198297500610352, -0.6201512813568115, 0.6096950173377991, 0.20385845005512238, 0.4033462703227997, 0.7036318778991699, -0.03406810015439987], [0.8433997631072998, 0.753788411617279, -0.7363230586051941, 0.33070117235183716, 0.6192669868469238, 0.5765236020088196, -0.693226158618927, 0.641408383846283], [-0.4258989691734314, -1.1973206996917725, -0.2139645367860794, -1.6305450201034546, -0.8052499294281006, 0.7870755195617676, -0.5555447340011597, 0.5977160930633545], [0.9883541464805603, -0.011848978698253632, 1.3864248991012573, -0.3119726777076721, -0.3392459750175476, -0.08460233360528946, -1.391748309135437, -0.07130104303359985], [-1.3693677186965942, 0.024210520088672638, -0.10629243403673172, 0.2695007920265198, -0.24067194759845734, 0.7811826467514038, -0.33158037066459656, -0.8477121591567993], [-2.5738632678985596, -0.28173840045928955, -0.5745775103569031, 1.2950332164764404, -0.0407455749809742, 1.2583438158035278, -1.4478251934051514, 1.2157140970230103], [1.1924442052841187, -1.2453354597091675, 0.8134022951126099, -0.9709539413452148, 1.256536602973938, 0.36783158779144287, -0.5255514979362488, -0.2495722770690918], [-0.23340991139411926, -2.7274270057678223, -0.6135960221290588, -1.1317987442016602, -0.039244089275598526, -1.6001781225204468, 1.3765331506729126, -0.8153760433197021], [1.8854315280914307, -1.7756427526474, 0.5172394514083862, -0.7441559433937073, -0.583049476146698, -1.818200707435608, -0.4394465386867523, 0.4293420612812042], [1.7194409370422363, -0.0979803055524826, 0.3625456690788269, -1.3484470844268799, 0.5578228235244751, -0.11671958863735199, -0.15775269269943237, 1.8618204593658447], [-0.23998405039310455, 0.044894393533468246, 1.004194736480713, 0.9927381873130798, -0.5725938677787781, 0.2667810916900635, -1.5378303527832031, 0.1058628112077713], [0.6825879216194153, 0.9182791113853455, 0.8046823143959045, 1.7542661428451538, 0.5455368161201477, -1.2879254817962646, 2.5413155555725098, 0.6188087463378906], [0.7152823805809021, 0.12415114045143127, -0.1638566255569458, 0.4577173888683319, 0.6848652958869934, -0.4270596504211426, 0.5501247644424438, -0.03799639642238617], [-1.0109397172927856, 0.37713974714279175, -1.462584137916565, -0.7588370442390442, -0.9489519596099854, 0.9727094769477844, -0.10601865500211716, -0.5915068984031677], [0.7110119462013245, -1.4191715717315674, 0.9331048130989075, 0.6153422594070435, 0.4550900459289551, 0.011979229748249054, 0.5881093144416809, -0.9101948738098145], [-0.37475061416625977, -0.655463457107544, 0.6951029896736145, 1.5605412721633911, -2.9884769916534424, 0.24025721848011017, 1.730141520500183, -0.652056872844696], [-0.37875479459762573, -0.795114278793335, -3.041813611984253, -0.6115940809249878, -0.6344005465507507, -0.34311720728874207, 0.08180844038724899, 1.3372033834457397], [-0.43436869978904724, 1.4451450109481812, -0.24000762403011322, -1.2403136491775513, 0.3377074599266052, 1.1011961698532104, -0.4729233384132385, 2.3813071250915527], [-1.1856988668441772, 2.6641197204589844, -0.014143994078040123, -1.5206623077392578, 1.1172951459884644, 0.6202581524848938, 1.2149879932403564, -0.34588438272476196], [-1.0801010131835938, -0.2734650671482086, -0.5221775770187378, 1.6870083808898926, 0.31752973794937134, -1.797171711921692, -1.4236918687820435, 1.463073492050171], [1.1491113901138306, -1.360837459564209, 1.2277133464813232, 0.014704459346830845, 0.6308974027633667, -1.0898773670196533, -0.6601343750953674, -1.5016844272613525], [1.028037428855896, -1.9807963371276855, 0.5534371137619019, 1.1399492025375366, 0.4571257531642914, -1.0868381261825562, -0.28017523884773254, 0.17733091115951538], [-0.02528379298746586, -0.03640081360936165, 0.5610621571540833, -0.1814226359128952, 0.6547968983650208, -1.2926510572433472, 0.6087551712989807, 0.8697108626365662], [-0.6136215925216675, 1.6909382343292236, -0.10203544050455093, -0.03281230479478836, -0.3027600646018982, -0.902862012386322, 0.5779181122779846, -1.6328046321868896], [1.0560487508773804, 0.39704227447509766, 0.4240185618400574, 1.279576063156128, -0.18815447390079498, 0.24882762134075165, 0.05259474366903305, 1.0747771263122559], [0.6576488614082336, 0.9309167861938477, 0.5499337315559387, 0.6847628951072693, -0.12991996109485626, -0.8010626435279846, 0.04281160980463028, 0.5115683674812317], [0.7439501285552979, 1.5721795558929443, 1.5865064859390259, -0.4746279716491699, -2.9995052814483643, -0.07085701078176498, -0.9431581497192383, -0.1280728429555893], [-0.7729506492614746, 2.487941026687622, -0.3718389868736267, 0.3754098117351532, -1.000441551208496, -0.1595376431941986, 0.28833863139152527, -0.41724708676338196], [0.711845338344574, -0.7474841475486755, -0.16682438552379608, 0.024531258270144463, -1.2082892656326294, 1.7805399894714355, -0.1547742336988449, 0.04541540890932083], [-0.9873207807540894, 0.5510994791984558, -0.006023051217198372, -0.6365906596183777, 0.2673409581184387, 0.506907045841217, -0.5205901265144348, 0.4333248734474182], [1.6054539680480957, -0.8289263248443604, 1.0276693105697632, -0.7510762810707092, 0.740225613117218, 0.3900023400783539, -1.1040304899215698, 0.02984737418591976], [0.3223731815814972, 0.3487786650657654, 0.13589297235012054, -1.4149173498153687, -0.0923331007361412, -0.6685636043548584, 0.25079184770584106, 1.4200313091278076], [-0.21106499433517456, 0.845643162727356, -0.4367358386516571, 0.33340656757354736, -0.1758541464805603, -0.030483154579997063, 0.009251553565263748, 0.4766632914543152], [-0.2118925005197525, -0.23702339828014374, -0.30469179153442383, 0.6634044647216797, -0.49590927362442017, 0.487531840801239, -2.618622064590454, 0.5503768920898438], [0.18399125337600708, 1.0045220851898193, -1.2172143459320068, -1.8132587671279907, -0.6133819818496704, -0.5997657775878906, 1.0126243829727173, 0.9287915229797363], [0.23147809505462646, 0.5598711371421814, -0.668855607509613, 0.06634841114282608, 0.5454308390617371, -1.6053600311279297, 1.2689597606658936, -0.026859741657972336], [-0.24641700088977814, -0.5765307545661926, -0.7658358812332153, 1.3285608291625977, 0.5252127051353455, 1.3601529598236084, 1.0723073482513428, 2.4492528438568115], [-1.1505067348480225, -0.31150394678115845, 2.13273549079895, -1.2100636959075928, -0.8013660907745361, -0.8203498721122742, 1.2995100021362305, -0.2557196319103241], [-0.6173444986343384, -0.6909571290016174, 0.8946624398231506, -0.15893873572349548, -1.0112398862838745, 0.15201526880264282, 1.2170045375823975, 2.0008254051208496], [0.16550737619400024, -0.12198911607265472, -0.6414411067962646, -0.9664655923843384, 0.7089617252349854, -1.5646775960922241, 1.2284899950027466, 1.0421499013900757], [1.0052976608276367, 0.4727323353290558, -0.5448367595672607, 0.8839803338050842, -0.818818986415863, -0.051470063626766205, 0.1242288127541542, -1.2545716762542725], [-1.52375066280365, -0.8164129853248596, -0.640791654586792, 1.1738841533660889, -0.7137653231620789, 0.19625285267829895, 0.3759799599647522, -0.7837362289428711], [-0.2322702556848526, -1.194670557975769, 0.050367020070552826, -1.413618803024292, 0.2517487406730652, -1.0328528881072998, 0.0075348359532654285, -0.6209498047828674], [-1.1252796649932861, -1.929701328277588, -0.31913092732429504, -0.36350345611572266, 0.5966349840164185, -0.42879718542099, -1.0834909677505493, -1.3284471035003662], [0.01989736221730709, 0.5866215825080872, -0.233059823513031, 0.15945129096508026, 0.10883761197328568, -0.932816743850708, -0.5493147969245911, 0.6769318580627441], [1.1653214693069458, 1.5451247692108154, 0.889481782913208, -0.2831852436065674, 0.3709172010421753, 0.2177562564611435, 0.9785087704658508, 1.3016772270202637], [-0.1760862022638321, -0.45203909277915955, -1.9157289266586304, -0.34089258313179016, -0.6327136158943176, -1.0839619636535645, 1.7304801940917969, 0.24234311282634735], [-0.4739902913570404, -0.02628384716808796, 1.0958200693130493, 0.046244122087955475, 3.0730104446411133, -1.6722966432571411, -0.7067393660545349, -0.15120165050029755], [-1.849442481994629, -1.053385615348816, -0.7664182782173157, -1.8200857639312744, 0.976495087146759, -0.1349431425333023, 0.9463687539100647, -0.3145109713077545], [-1.626172423362732, 0.20846788585186005, -1.3819079399108887, 0.38996952772140503, -0.1013910323381424, 2.5336155891418457, -0.4568047523498535, 0.2755826711654663], [-0.5743971467018127, 1.052528977394104, 0.9496831297874451, 0.3587920367717743, -1.8601760864257812, 1.2595231533050537, -0.9269847869873047, -0.832190752029419], [-0.3874446749687195, -0.6008397340774536, -0.6429058909416199, 0.4201688766479492, -0.10143402218818665, 0.8958252668380737, 0.580780565738678, 1.2264299392700195], [0.5228645205497742, 0.17036142945289612, 0.1084221601486206, -0.5522473454475403, 0.8669673204421997, -1.6946394443511963, 0.7338337302207947, 0.25258827209472656], [-1.490369200706482, -0.9255403280258179, -1.6094272136688232, -0.0040288567543029785, 2.1040050983428955, -0.09132835268974304, 0.8276404142379761, -0.8816408514976501], [1.3827415704727173, 1.2283880710601807, 2.3561553955078125, 1.2621647119522095, 1.3283501863479614, -0.7579533457756042, -0.6089736223220825, 0.4588480591773987], [0.02185158059000969, -1.1243305206298828, -0.6510381102561951, 0.8511555790901184, -0.6725187301635742, 1.56136155128479, -1.4088108539581299, -1.1772878170013428], [0.12520617246627808, 0.4084317088127136, 1.7182832956314087, -1.212969422340393, -1.6592501401901245, -0.8077018857002258, 0.2510957717895508, 0.8031343817710876], [0.5791970491409302, 0.33169037103652954, 1.7213070392608643, -0.6222869753837585, 0.4687751233577728, -0.26377832889556885, -0.2682102620601654, -1.3828346729278564], [-0.24301929771900177, -0.7261354327201843, -0.3224778473377228, 0.22809474170207977, -1.3533071279525757, 0.36921238899230957, 1.1585246324539185, 0.46535032987594604], [-0.24671652913093567, 1.3551565408706665, -0.8288372159004211, -0.07590228319168091, 1.5242983102798462, 0.9905869364738464, 0.3144702911376953, 0.12309173494577408], [-0.29553309082984924, -0.19631430506706238, -0.40870338678359985, -0.7732446193695068, 1.4606339931488037, -0.37514713406562805, -0.3495158553123474, 0.9702529311180115], [1.2397197484970093, 1.6169863939285278, -0.23544353246688843, -0.1019776239991188, 0.5564287304878235, 0.6160146594047546, 0.09373930096626282, 0.1684839427471161], [-2.178696632385254, -0.46204110980033875, 0.16554032266139984, 1.3004614114761353, 1.362127423286438, -0.6684619188308716, -1.0207791328430176, 0.42988091707229614], [1.3495639562606812, 0.003738188650459051, -1.5698801279067993, -0.4758774936199188, 0.17752614617347717, 2.3615574836730957, 1.1563735008239746, 1.4174485206604004], [0.9806144833564758, -0.7741418480873108, 0.08796198666095734, 0.6424298882484436, 0.9499463438987732, 0.7737336158752441, 1.108988642692566, -0.5573720932006836], [-0.9225369095802307, -0.09081776440143585, -0.4706919193267822, 1.6083695888519287, 0.2109655886888504, -0.1878734976053238, -1.2089897394180298, -1.6951957941055298], [2.352855920791626, 0.43728432059288025, -0.7947864532470703, -0.45442456007003784, -2.9123332500457764, 0.8233930468559265, 0.9630124568939209, -1.0338844060897827], [0.9508525729179382, 0.7367770075798035, 0.5057782530784607, -2.1377406120300293, 0.07345308363437653, 0.6674544811248779, -2.6279079914093018, 0.5957284569740295], [-0.342776358127594, -0.06744101643562317, 0.551462709903717, 0.8905335664749146, 0.733429491519928, -0.4373943507671356, 0.3698451817035675, 0.1192471981048584], [1.7786141633987427, 2.0624279975891113, -0.08794530481100082, 1.5712345838546753, -1.1185110807418823, -0.4088194966316223, 2.9072377681732178, 1.2160207033157349], [0.5655978918075562, 1.0911972522735596, -0.8460800051689148, -0.24354198575019836, 0.5510686635971069, 0.45007672905921936, 0.6981039047241211, 0.6476631164550781], [-0.2557515799999237, -0.32739850878715515, -0.5072495341300964, -2.7166783809661865, 0.6067299842834473, -1.2654894590377808, -1.2561477422714233, -0.04338627681136131], [0.19327671825885773, -0.1485832780599594, 0.9215255975723267, -1.7189245223999023, 0.8562259078025818, 0.19047129154205322, -0.5418276190757751, 0.6291272044181824], [-0.15797118842601776, 0.2933815121650696, -0.4406997263431549, -2.082035541534424, -2.4562103748321533, 0.6088326573371887, 0.07797765731811523, -1.7018407583236694], [0.3440151810646057, 1.928057312965393, -3.646383285522461, 0.6275460124015808, 0.25353899598121643, -0.2162286341190338, -0.3346472382545471, -0.6705577969551086], [-0.38960954546928406, 0.3953460454940796, -0.26114755868911743, -1.6169506311416626, 0.42849284410476685, -0.2722712457180023, -0.7264434099197388, -0.2415543794631958], [0.5214167237281799, -0.6940065026283264, -0.7057769298553467, -2.034045696258545, 0.14800521731376648, 0.16609331965446472, 0.017320264130830765, 0.45290786027908325], [3.0859873294830322, -1.8133233785629272, -0.1996205747127533, -0.6217076778411865, -0.9151198267936707, 1.4870705604553223, 0.5199082493782043, -0.8525422811508179], [0.15916018187999725, -0.5409398078918457, 2.335566759109497, 0.03942369669675827, 1.700084924697876, 0.7673683762550354, 1.0664726495742798, -0.5454893112182617], [2.2521822452545166, -2.1740031242370605, -0.9950198531150818, -0.7766912579536438, -0.031301312148571014, -0.47806820273399353, 0.29932618141174316, 0.5334200263023376], [-1.8320858478546143, 0.994857132434845, 0.2720867991447449, 0.24208489060401917, 1.3561391830444336, -0.6274566054344177, 0.18580792844295502, 1.1337674856185913], [-0.6812971830368042, 0.46606603264808655, 1.3637775182724, -0.03275166451931, -0.5023665428161621, 1.0416457653045654, -0.6397464871406555, -1.1517398357391357], [0.6852307915687561, -1.2309664487838745, 0.102530837059021, 1.340901255607605, -0.08900976181030273, -0.028034983202815056, 0.5534605383872986, -0.18840216100215912], [0.9610251188278198, -0.7854443192481995, -0.721024751663208, -0.5554687976837158, 0.5044736266136169, 1.943782925605774, 2.6400814056396484, -0.8220414519309998], [0.972749650478363, -0.31512752175331116, 0.23188695311546326, -0.5922015309333801, -0.6191376447677612, -0.45566320419311523, -1.2932018041610718, 0.9008610844612122], [0.2710401713848114, 0.12097728252410889, -0.7589552402496338, 0.6146038770675659, 0.4847136437892914, -0.36771148443222046, -0.2339710295200348, -0.06895336508750916], [0.7460420727729797, -0.11287185549736023, 1.8606880903244019, -0.23742513358592987, 0.6540504693984985, -0.872912585735321, 0.43658220767974854, 0.7498937249183655], [1.5218960046768188, 0.1791726052761078, 0.03376566618680954, -1.3149060010910034, -1.1335078477859497, 0.4301122725009918, -0.45576849579811096, 0.7354533076286316], [2.147580862045288, 0.9624276161193848, -0.5241925716400146, -0.25622251629829407, 0.10999522358179092, -0.7350857853889465, -0.13791972398757935, -1.3598607778549194], [1.2729216814041138, -1.0153627395629883, 1.0827454328536987, -0.09943845868110657, -0.049479126930236816, 1.2194041013717651, -0.47756725549697876, -0.7345470190048218], [-2.0118448734283447, 0.0721888616681099, -0.7081434726715088, -0.37993213534355164, -0.5983580946922302, 1.4213104248046875, -1.7390345335006714, -0.3815070688724518], [0.15364040434360504, -2.091120958328247, 1.3678656816482544, 0.8766700625419617, -0.38740065693855286, 1.0841395854949951, 1.299129843711853, -1.8485448360443115], [0.22718515992164612, -0.5280473828315735, 1.6053279638290405, -0.29590684175491333, -1.0033358335494995, 0.5214439630508423, 0.038541991263628006, -0.5581493377685547], [-1.6973986625671387, -0.23720456659793854, 0.3034067451953888, -1.5255461931228638, 0.8585100769996643, -0.18317773938179016, 0.655582070350647, -0.999895453453064], [-1.0119112730026245, 0.005626744125038385, -1.535007119178772, -0.3619943857192993, -1.1933701038360596, -0.8289826512336731, 0.35208532214164734, 1.7449568510055542], [0.3627384603023529, -0.5388125777244568, 0.9085321426391602, 0.6153612732887268, -1.5689220428466797, 0.3303981125354767, 0.535760760307312, 0.37164250016212463], [-0.9214398264884949, -0.1998578906059265, 0.3498803973197937, 0.875204861164093, 0.5466601848602295, 0.02121555805206299, 0.4919358789920807, -0.6402491927146912], [0.7377257943153381, 0.07581520825624466, -0.7913213968276978, 0.68468177318573, -0.8763818144798279, 0.33719339966773987, -0.1804867386817932, -0.8741969466209412], [-1.0704792737960815, 0.6536397337913513, -0.917790412902832, 0.5974849462509155, 1.0954933166503906, -0.7098820805549622, 0.2611044943332672, 0.4552181661128998], [0.5752840042114258, 2.6424756050109863, -0.8104353547096252, -1.6620724201202393, -0.07633587718009949, 0.5330529808998108, 1.0045284032821655, -0.01992897316813469], [-0.1114872395992279, -0.5132520794868469, 0.21396712958812714, -0.01695886440575123, 1.3677868843078613, 0.8064380884170532, 0.21633876860141754, 0.5078577995300293], [1.554946780204773, -0.6996679306030273, -0.2451988309621811, 0.14959822595119476, -0.30801841616630554, -0.2068624198436737, 1.8451203107833862, -0.9291003942489624], [-0.6820762157440186, 1.0334354639053345, -0.545914351940155, 0.9174328446388245, -1.0962955951690674, -0.5492466688156128, 0.31963926553726196, 0.5908272862434387], [-0.7812289595603943, -0.0006924363551661372, -0.6053171753883362, -1.8760420083999634, -1.2760212421417236, -2.3682520389556885, -1.2443645000457764, -0.12091527134180069], [-0.42253562808036804, -0.5326854586601257, -0.18786272406578064, 1.3614065647125244, -1.0506455898284912, 0.29545271396636963, -0.008370568044483662, 1.0009692907333374], [1.445560097694397, 0.3365512490272522, 0.7164299488067627, 0.8510740399360657, 0.30667775869369507, -0.5367104411125183, -0.29687339067459106, 0.37220498919487], [1.107770562171936, -0.08549242466688156, -0.09926599264144897, -1.3419429063796997, -0.32569125294685364, -1.5387555360794067, 1.980282187461853, 1.2818433046340942], [-0.5204681158065796, 0.09467703104019165, 1.0660243034362793, 2.185028553009033, -0.8047422766685486, 0.631544291973114, 0.12168838083744049, 0.8141267895698547], [-0.38839206099510193, 1.8978769779205322, -0.6163029670715332, -0.7996615767478943, 0.06469537317752838, -0.12877558171749115, -0.5457587838172913, -0.504565954208374], [0.19016584753990173, -1.462697148323059, -0.38683080673217773, -1.0670361518859863, -0.6475410461425781, -0.3693650960922241, 0.49729758501052856, 0.25071921944618225], [-0.012635289691388607, -1.3278292417526245, -0.3618839979171753, -1.7264519929885864, 0.7722419500350952, -0.04737715795636177, 0.5733880400657654, -2.2594661712646484], [-0.3971514403820038, -0.6560940146446228, 1.188966155052185, -0.9667336940765381, -0.14099013805389404, 1.6467339992523193, 0.8947060108184814, 0.15121643245220184], [0.136562317609787, -0.9111592769622803, -0.49892643094062805, 0.37475305795669556, 0.890163779258728, -1.507608413696289, -0.1176973357796669, -0.024224525317549706], [0.4888222813606262, -1.8648205995559692, 0.9492988586425781, 0.8289737105369568, 0.7621864080429077, -1.9336957931518555, 1.1110703945159912, 0.8931744694709778], [0.27846693992614746, -0.9224399924278259, 0.6757916808128357, -0.06948711723089218, 0.9146401286125183, -0.6327826380729675, 1.2213777303695679, 1.19767427444458], [1.6983460187911987, 0.39369699358940125, 1.5912262201309204, 0.8754065036773682, 0.468038409948349, -0.5613455176353455, -1.6854350566864014, -0.1545512080192566], [-1.4440187215805054, -1.057550311088562, -1.6881728172302246, -1.3439159393310547, 1.3364629745483398, -0.5862758159637451, 0.8111699223518372, -0.13703566789627075], [-1.0096571445465088, -2.1504995822906494, -1.0901093482971191, 0.5835879445075989, -1.0373064279556274, 2.7333686351776123, -1.2819972038269043, 0.38678672909736633], [-0.04785386100411415, 0.297140508890152, -0.175027996301651, 0.6308535933494568, -1.1391831636428833, -0.7673720717430115, 0.795102059841156, -0.11278115212917328], [-1.9812965393066406, -0.8194831609725952, -0.5696008801460266, 2.966017961502075, 0.2934616208076477, -1.889559268951416, 1.3208204507827759, -2.0208945274353027], [-0.7679685950279236, -0.8735349178314209, 0.15456253290176392, 0.9277178645133972, -0.9870839715003967, 2.225921154022217, 0.5029664039611816, -1.606355905532837], [-0.6604821681976318, 2.4682106971740723, -1.2420564889907837, 0.4402504861354828, -0.27044612169265747, -0.6012058854103088, 0.27296119928359985, 0.07588103413581848], [-1.5715194940567017, 0.583584725856781, -0.8504904508590698, -0.8626601696014404, 0.2780023515224457, 0.6461221575737, 0.7367362976074219, -2.095966339111328], [0.9852147698402405, 0.4214535653591156, -1.4079657793045044, -0.3960179388523102, 1.881564974784851, -0.6252825856208801, 0.690220057964325, 0.142647847533226], [0.7072579264640808, -0.1250341683626175, -1.8813191652297974, 0.23326490819454193, 0.9431881308555603, 0.23909534513950348, -1.3549162149429321, -0.11481447517871857], [-0.5531352758407593, -0.7181406021118164, 0.6002616882324219, 1.7477271556854248, -1.6988331079483032, -0.024717628955841064, 2.1737732887268066, -0.6563757658004761], [-0.1127157136797905, 2.04317307472229, 1.4414434432983398, 0.6621227264404297, 0.03174750506877899, -0.6933631300926208, 1.077059268951416, -0.04406441003084183], [-0.17986734211444855, 0.7151148915290833, -0.004312017932534218, -0.38260847330093384, -1.19960355758667, -1.9098153114318848, 0.6209484934806824, -0.10238657891750336], [1.0027440786361694, -2.0078625679016113, 0.08695585280656815, 0.17407728731632233, 0.944069504737854, 0.8139680027961731, -0.042905043810606, 1.74626624584198], [1.3532633781433105, 1.271863341331482, 1.1747606992721558, 0.033018603920936584, -2.0338082313537598, 0.36723875999450684, -1.567568302154541, 1.469409465789795], [-1.6427232027053833, 1.0074496269226074, -0.04896434396505356, 0.13463501632213593, -0.9944908618927002, 2.195594549179077, -0.6946337819099426, 0.2948457598686218], [-0.22832798957824707, -0.9857662320137024, 0.4795789122581482, 0.2705838084220886, -0.19248194992542267, 1.7239320278167725, 0.18552863597869873, -1.1192318201065063], [-0.5579977035522461, 0.5279721617698669, -1.3188751935958862, -0.3876895010471344, -1.0626353025436401, 0.29005497694015503, 1.3685312271118164, -0.44628989696502686], [-1.8271963596343994, 0.21422050893306732, -1.1306533813476562, 0.9149401187896729, 0.3475361168384552, -1.1848247051239014, -0.2805067300796509, 0.6874285936355591], [-0.45842501521110535, 0.041113924235105515, 0.8227875232696533, -0.07849199324846268, -0.6891010999679565, -0.7089765667915344, 1.2674510478973389, -0.018617896363139153], [0.35890376567840576, -0.7587856650352478, 2.0432398319244385, -1.374240517616272, 0.19255416095256805, 1.280564785003662, 0.34676334261894226, 0.7458973526954651], [-0.8371214866638184, 1.9486708641052246, 1.3149887323379517, 0.8621126413345337, 3.205854892730713, -0.23869122564792633, 0.11576338112354279, 0.8602980375289917], [-1.3978127241134644, 1.0811630487442017, -0.40842247009277344, 1.9114147424697876, -0.9320237636566162, -2.2278943061828613, 0.9877902269363403, 0.1845969706773758], [1.3592438697814941, 0.677650511264801, -0.5498258471488953, 2.8045156002044678, 0.6374300718307495, 0.4461861550807953, 0.27032041549682617, 0.08697416633367538], [1.985813021659851, 1.9556043148040771, -0.7626497149467468, -0.07054102420806885, -0.25366055965423584, -0.7895184755325317, -0.2400222271680832, 0.16580168902873993], [0.9472194314002991, -0.7321400046348572, -0.7732776999473572, -1.9881433248519897, -0.2690691351890564, -1.6750017404556274, -0.7473854422569275, -1.4148871898651123], [0.4633086323738098, 0.4234810769557953, 0.15704604983329773, 0.5006348490715027, 0.917682945728302, 0.8809357285499573, -1.3021348714828491, 1.6123024225234985], [0.6577328443527222, -1.8775498867034912, -0.8215345144271851, 0.33565330505371094, -0.8805349469184875, 0.5684415698051453, 0.12199317663908005, 0.8866162896156311], [1.4007924795150757, -0.1670561134815216, 0.6191201210021973, 0.3061274290084839, -0.02780984528362751, -0.1331252157688141, -0.5341290831565857, -0.6607625484466553], [-1.7179425954818726, 2.243581771850586, -0.059488940984010696, 2.153646230697632, -0.45717939734458923, 1.3474655151367188, 1.4245525598526, 0.3986486494541168], [1.6877315044403076, 0.896712064743042, 1.075204610824585, -2.1746978759765625, -0.2660056948661804, -0.7392326593399048, 1.7730456590652466, -1.8727363348007202], [-1.6787983179092407, -1.1348521709442139, 0.4688279628753662, 1.675209879875183, -0.3988015651702881, 0.8316473364830017, -0.009796682745218277, 0.5946887135505676], [0.22352808713912964, -0.4755377173423767, -0.15993303060531616, -0.1665695309638977, -1.899480938911438, 0.4701750576496124, 1.0778130292892456, 1.8389321565628052], [0.09782421588897705, 0.010248116217553616, -0.5356059074401855, 1.1686530113220215, -1.001050591468811, 0.48976069688796997, -0.10770723223686218, 0.6378560066223145], [0.4838123917579651, 0.6206671595573425, 0.4970192015171051, -0.6986645460128784, 1.2864660024642944, -0.581662118434906, -0.4661073088645935, 0.4052431285381317], [-1.2723257541656494, 1.3533670902252197, -1.9641629457473755, -0.28253111243247986, -1.008435845375061, -0.6608715653419495, 1.1275200843811035, -0.6632491946220398], [0.6175239682197571, 0.3747386932373047, -0.08734264224767685, -1.6958537101745605, 0.5998396277427673, 0.3464809060096741, -0.8178759813308716, 0.6103785037994385], [0.3822830319404602, -0.2834344804286957, 0.031297653913497925, -0.1811293065547943, -0.9688237905502319, 1.0564264059066772, -1.3656907081604004, -1.1651686429977417], [-0.290399432182312, -0.7336518168449402, 0.44531136751174927, 0.3837604820728302, 0.2182234674692154, -0.5539525747299194, -1.308644413948059, 1.0258967876434326], [-0.3477945625782013, 0.31907275319099426, -1.020552635192871, 1.4928094148635864, 0.5828962922096252, 0.3826081156730652, -4.99130192110897e-06, 0.8356978893280029], [-0.941302478313446, 0.1316681206226349, 0.004800340626388788, -0.35426533222198486, -1.555602788925171, 1.2449638843536377, 0.11977839469909668, 1.5520228147506714], [1.140248417854309, 1.8428456783294678, 0.7303664684295654, 0.4109291434288025, 2.3175132274627686, 2.842346429824829, 0.11094266921281815, 0.6022139191627502], [1.072853446006775, 1.593121886253357, 0.4955708682537079, 0.47858762741088867, 0.12444622069597244, 2.619117498397827, 0.1098056435585022, 0.23182158172130585], [-0.6206421256065369, 1.393967628479004, 0.06246666610240936, -0.9400607943534851, 0.17284055054187775, -1.3818875551223755, -0.1487981528043747, -0.1522059440612793], [-0.5319564938545227, 0.034649141132831573, 0.4343132972717285, -0.9500914812088013, 1.7429649829864502, 2.791496515274048, -0.5243244767189026, -1.485211968421936], [-0.8155461549758911, 0.07508682459592819, 0.47800213098526, 0.7246175408363342, -0.16830213367938995, 1.0292882919311523, -1.0500061511993408, 0.5490468144416809], [1.3563015460968018, -0.9551497101783752, -0.3668025732040405, 0.22518938779830933, -0.9190317988395691, 1.0996431112289429, 0.47402480244636536, -1.2468053102493286], [1.2007197141647339, 1.80012047290802, -0.4444686472415924, -0.5293759107589722, 0.7810843586921692, -0.19277751445770264, -0.5892459154129028, 0.46366265416145325], [-0.08756804466247559, -0.3489737808704376, -0.32643812894821167, -0.2061205357313156, -1.1921679973602295, -0.4057457745075226, -0.4911128282546997, 0.389150470495224], [-0.6529092192649841, -1.032372236251831, 0.829239010810852, 0.1212267354130745, 0.2751321792602539, -0.4386899769306183, -0.39538881182670593, -0.3668104410171509], [-1.4173407554626465, 0.119948610663414, -0.09975643455982208, 1.2825367450714111, 1.75347900390625, 0.7445341348648071, 0.014097141101956367, 0.6469372510910034], [2.2611610889434814, -0.7312763929367065, 0.5843393206596375, 0.5107499957084656, -1.652207612991333, -1.2445021867752075, -2.0324344635009766, -0.006689090747386217], [2.1897342205047607, 0.08121543377637863, 0.653417706489563, 2.1846232414245605, 0.0005263875937089324, 1.4509013891220093, -0.5098444819450378, -1.5207204818725586], [0.05609404668211937, -1.6263914108276367, 0.16893094778060913, 0.30698785185813904, -1.6271991729736328, 0.6425821185112, -1.1627047061920166, -1.0349676609039307], [1.0880556106567383, -1.779098391532898, -1.004172921180725, -0.7656380534172058, 0.5415242910385132, 0.8937953114509583, 0.18631967902183533, -0.568386435508728], [-1.5860871076583862, -0.050408415496349335, 0.028719624504446983, 0.6764249205589294, -0.20453791320323944, 0.11683149635791779, 0.9115602374076843, -1.9290800094604492], [0.9498528242111206, -0.7838900089263916, 0.36215537786483765, 1.1455585956573486, -0.6181869506835938, -0.11454424262046814, -1.2642854452133179, -0.8330309391021729], [-1.0083597898483276, -0.5362944602966309, -0.6115505695343018, -1.4301958084106445, 1.3390463590621948, -0.40237605571746826, 0.2012699693441391, -1.1040419340133667], [0.9354395866394043, -0.17156633734703064, 0.03940017148852348, -0.03538290411233902, 0.889371395111084, -0.6833318471908569, 0.6000686883926392, 0.0515124686062336], [-1.2544023990631104, -0.63474440574646, -1.1018362045288086, 0.019847452640533447, 0.47073614597320557, 2.5463404655456543, 0.081680066883564, -0.28228282928466797], [0.5626208782196045, 1.2166571617126465, -0.16473080217838287, 0.36382269859313965, 0.6236314177513123, -0.34738099575042725, -1.0490657091140747, 0.42075684666633606], [-0.8980246186256409, 0.24768131971359253, 0.46711766719818115, -0.9127563834190369, -0.31909701228141785, -0.7570188045501709, -0.5996631979942322, 1.9775350093841553], [1.63361394405365, 0.5610228180885315, 0.24342471361160278, 0.7031792402267456, 2.1933891773223877, 2.68485689163208, 0.7333000898361206, -1.329685926437378], [-0.13042838871479034, 0.5449627637863159, -1.1701765060424805, -0.01622883975505829, -0.5435630679130554, 0.5853349566459656, 2.1797375679016113, 0.023100238293409348], [1.0876049995422363, 0.37714576721191406, -2.017129898071289, 0.12461631000041962, -0.5937477350234985, 0.9003349542617798, -0.21991434693336487, -0.41760092973709106], [-0.11509008705615997, -0.4770306646823883, -1.060433268547058, 1.9877517223358154, -2.3556835651397705, -0.23209621012210846, -0.24029551446437836, 0.8530930876731873], [0.5159286260604858, -1.2722904682159424, -0.6433835625648499, -1.0732959508895874, -1.1784535646438599, 1.4343583583831787, 1.4880971908569336, 0.5317612290382385], [-0.691781222820282, 0.8603777885437012, -1.599987506866455, -1.2025816440582275, -0.013502984307706356, 0.5065587759017944, 0.08797836303710938, -1.8603763580322266], [0.4708300232887268, 1.1455861330032349, 1.1715660095214844, 0.7042254209518433, -1.678159475326538, 1.6500036716461182, 1.7521663904190063, -0.2784985601902008], [1.8735541105270386, 0.8793186545372009, -0.22580713033676147, 0.4133882522583008, -0.36009886860847473, -0.06925033777952194, 0.6265191435813904, -0.6876837015151978], [0.07232082635164261, -0.235920250415802, -0.18947985768318176, -0.46708038449287415, -0.0948929637670517, 0.9320245981216431, -0.5204246640205383, -0.26133596897125244], [-0.08813979476690292, 1.2448313236236572, 0.25554636120796204, -0.08819365501403809, 0.8016899824142456, -0.8644577860832214, -0.30904608964920044, 0.8790199756622314], [-0.030011247843503952, 0.5766654014587402, 0.5945287346839905, -0.30458733439445496, -0.15618449449539185, -0.11083311587572098, -0.5408675074577332, -1.5829206705093384], [-2.932227611541748, 0.6921347379684448, -1.0128984451293945, -0.41508451104164124, -0.4324607253074646, 0.376977801322937, -1.1559617519378662, 0.1511644423007965], [0.5878134965896606, -0.0014163196319714189, 0.020599128678441048, 0.5079149007797241, 1.2438569068908691, -1.2161120176315308, 2.1174633502960205, 0.2106974869966507], [0.1590847223997116, 1.828800082206726, -1.07906174659729, -1.193406105041504, -1.0127333402633667, -1.154320478439331, -1.3055857419967651, -1.1321957111358643], [0.9753618240356445, -1.1372015476226807, 0.30977293848991394, -0.18854789435863495, -0.7403535842895508, -0.9016643762588501, 0.6205554008483887, -0.11584596335887909]], "dtype": "float32"}, 4, 32], "e": {"__tup__": [{"__nd__": [[12.166851997375488, -70.6104965209961, -9.455381393432617, 60.48308181762695, -41.72782516479492, 7.3873443603515625, -23.78207015991211, 37.05414962768555], [3.502154588699341, 3.70443058013916, 32.10929870605469, 12.322637557983398, -44.14237976074219, 77.85835266113281, -24.44542121887207, -2.39677095413208], [-23.972076416015625, -25.34654998779297, -86.44571685791016, -15.025873184204102, 29.814828872680664, -32.04573059082031, -13.29846477508545, 12.899726867675781], [-32.10201644897461, -70.76887512207031, 13.062997817993164, 21.22037696838379, -42.98935317993164, -34.2604866027832, -0.21431878209114075, 0.1866253912448883], [8.204028129577637, 8.897055625915527, 95.07330322265625, 12.564709663391113, -17.935810089111328, -36.42300796508789, -35.40617752075195, -24.567800521850586], [-42.009674072265625, -31.42637062072754, -3.724755048751831, 25.62611198425293, -18.398887634277344, -29.29535484313965, -30.277252197265625, -32.78993606567383], [-38.113746643066406, 4.804358959197998, -28.61428451538086, -4.107079982757568, 10.405040740966797, -25.247058868408203, 53.1973876953125, -4.109438896179199], [36.00223159790039, 18.504064559936523, 35.82234573364258, -33.70753860473633, 27.750232696533203, 8.845160484313965, 15.37937068939209, -16.757619857788086]], "dtype": "float32"}, {"__nd__": [[[-7.360341548919678, -24.769140243530273, 5.48776912689209, 9.351264953613281, -37.26194763183594, 3.1835412979125977, -6.333577632904053, 36.6400146484375], [-5.425123691558838, -3.2977776527404785, 15.573373794555664, -12.73989486694336, -6.040367126464844, 11.994698524475098, -3.179788589477539, -0.08157585561275482], [-13.545143127441406, 0.23489312827587128, -32.67448425292969, -10.995804786682129, 7.7615790367126465, -4.638885974884033, -33.05926513671875, -9.422057151794434], [4.709527492523193, -15.408178329467773, 4.598163604736328, -2.4367072582244873, -11.997238159179688, -7.35921573638916, -30.056264877319336, 7.777980804443359], [14.016550064086914, -8.266508102416992, 44.151824951171875, 5.700236797332764, 10.815793991088867, -14.07159423828125, -18.600387573242188, 7.474243640899658], [-4.861881256103516, 17.94354248046875, -12.607738494873047, -11.009223937988281, -19.150127410888672, -9.758511543273926, -18.641817092895508, 12.072755813598633], [-22.251327514648438, 14.293554306030273, -9.611145973205566, 5.1003546714782715, -14.427094459533691, -5.519055366516113, 5.484598636627197, -4.265636444091797], [3.5661611557006836, 26.94410514831543, 14.691642761230469, -1.5319236516952515, 2.1925477981567383, 6.483108043670654, -33.080692291259766, 4.528302192687988]], [[23.486743927001953, -26.981117248535156, 8.570686340332031, 35.50830078125, 10.334851264953613, -1.2264119386672974, -7.979743480682373, -9.57624626159668], [8.378870964050293, -11.5856294631958, 11.728714942932129, 2.992750883102417, 8.309475898742676, 20.28322410583496, -1.2743403911590576, -9.503345489501953], [-0.7318257689476013, -9.329876899719238, -26.2004451751709, -17.204967498779297, 14.674829483032227, -1.3209408521652222, -0.7230761051177979, 29.423301696777344], [-9.004334449768066, 3.0205235481262207, -12.742830276489258, 18.09083366394043, -10.37094783782959, -7.1237993240356445, 11.325181007385254, 1.8576464653015137], [-17.374799728393555, -7.821542263031006, 7.918476581573486, 17.16529655456543, -21.955734252929688, -28.12702178955078, -6.230347633361816, 26.357439041137695], [-6.529084205627441, -10.568044662475586, 10.942403793334961, 7.546370983123779, 11.320884704589844, 2.0082848072052, -11.651884078979492, -4.3184123039245605], [-19.730022430419922, 4.323043346405029, 4.691979885101318, 23.747549057006836, 21.87582778930664, 4.528665542602539, -8.182086944580078, 7.344740390777588], [15.638418197631836, -33.65584182739258, 1.4154940843582153, -0.3838302791118622, 3.323241949081421, -2.4922702312469482, 16.853713989257812, -4.834878444671631]], [[-7.196107864379883, -10.014703750610352, 0.629404604434967, 3.7458102703094482, 3.861422300338745, -0.4783245325088501, 3.7949936389923096, -7.6217851638793945], [-12.702686309814453, 17.69792366027832, -11.8967866897583, -4.113195419311523, -17.904237747192383, -4.653059482574463, -9.308116912841797, 11.909846305847168], [5.915116786956787, -34.12331008911133, -16.476743698120117, -15.885597229003906, 9.17124080657959, -34.07246780395508, 14.738517761230469, -6.446543216705322], [-17.66986846923828, -25.668399810791016, 17.46538734436035, 0.8018457293510437, -5.993525981903076, 7.7197418212890625, 15.424840927124023, 3.91184663772583], [5.330562591552734, -3.543619155883789, 21.584514617919922, -26.682655334472656, 4.428269863128662, -6.557134628295898, -13.959403991699219, -25.90934944152832], [-8.207876205444336, -20.510839462280273, 1.0029640197753906, 27.61693572998047, -20.2659854888916, -6.081336975097656, -20.765003204345703, -6.370612144470215], [3.595618724822998, -20.807235717773438, -5.63499116897583, -23.9963321685791, 0.24955424666404724, -30.807971954345703, 28.16202163696289, 12.03011417388916], [13.925813674926758, -0.5989888906478882, 13.023619651794434, -2.046797275543213, 6.732380390167236, -17.374818801879883, 26.57839012145996, -11.08923053741455]], [[3.236558198928833, -8.845538139343262, -24.14324188232422, 11.877706527709961, -18.66214942932129, 5.908539295196533, -13.263742446899414, 17.612165451049805], [13.251093864440918, 0.8899133801460266, 16.703996658325195, 26.18297576904297, -28.507251739501953, 50.233489990234375, -10.683175086975098, -4.721696376800537], [-15.610223770141602, 17.87174415588379, -11.09404468536377, 29.060495376586914, -1.7928211688995361, 7.986565113067627, 5.745360851287842, -0.6549744009971619], [-10.137341499328613, -32.712825775146484, 3.7422780990600586, 4.764404773712158, -14.627641677856445, -27.49721336364746, 3.0919241905212402, -13.360848426818848], [6.231714248657227, 28.528724670410156, 21.418485641479492, 16.381832122802734, -11.224140167236328, 12.332740783691406, 3.383962631225586, -32.490135192871094], [-22.410831451416016, -18.291030883789062, -3.0623841285705566, 1.472029685974121, 9.696340560913086, -15.463791847229004, 20.781452178955078, -34.17366409301758], [0.27198365330696106, 6.994996547698975, -18.060129165649414, -8.958651542663574, 2.706753730773926, 6.551302909851074, 27.732852935791016, -19.218656539916992], [2.871837854385376, 25.814788818359375, 6.691590309143066, -29.744985580444336, 15.502062797546387, 22.229141235351562, 5.027958393096924, -5.361813545227051]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.9612677097320557, -0.02740432694554329, 2.7979228496551514, -1.3061109781265259, 0.01740924082696438, 0.9928523302078247, 0.6748734712600708, 1.0140758752822876, -0.37330976128578186, -1.1052626371383667, 2.588113784790039, -2.145411968231201, -0.4678625166416168, -1.3520013093948364, -2.042689085006714, -0.07487618923187256, 1.0085405111312866, -0.240152969956398, 0.3198818266391754, -2.3098855018615723, -1.1652265787124634, -0.1030862033367157, 0.3594318926334381, 0.1797354519367218, 1.4802937507629395, -0.7054319977760315, 0.035795658826828, -1.1728923320770264, -1.2475531101226807, -0.9346947073936462, 0.6959055662155151, -0.2959751486778259, 0.02816811017692089, 0.8880131840705872, -0.6545118093490601, -1.0802212953567505, 0.9634748101234436, 2.009633779525757, 1.3201673030853271, 0.5025981664657593, 1.3155860900878906, 1.005424976348877, 1.4782836437225342, -0.11017537117004395, 2.183588981628418, 0.17818421125411987, 1.1833690404891968, 1.765934944152832, 1.5800814628601074, 1.588168978691101, 1.6019059419631958, 0.3160308301448822, 1.2130979299545288, -0.3929506540298462, -0.34130504727363586, -0.859263002872467, -0.46501484513282776, 0.020520348101854324, -0.9292500615119934, 0.7897677421569824, -0.34645575284957886, -0.7842081785202026, 0.029405346140265465, 0.8405341506004333, -0.9319764971733093, -0.27541497349739075, 2.406822443008423, -0.07408387959003448, 0.9243771433830261, -1.2721627950668335, -0.45781585574150085, -0.9108631610870361, 0.4537131190299988, -0.5887289643287659, 1.570002555847168, -1.8293179273605347, -1.2984037399291992, -1.7459639310836792, 0.12870462238788605, 1.0466187000274658, -0.9931159019470215, -0.770219087600708, 1.573776125907898, 0.5554185509681702, -0.8734275102615356, -1.3713537454605103, 2.314969539642334, -0.7609672546386719, -0.07316701114177704, 0.14547334611415863, 0.5142480134963989, -1.261600375175476, -0.6425105333328247, 0.49648964405059814, -1.126327633857727, 1.9888979196548462, 1.1643139123916626, -1.2111107110977173, 0.9138548970222473, 0.5784264802932739, 0.5970129370689392, 0.200219064950943, 0.13599519431591034, 0.022766491398215294, -1.467330813407898, -0.8628329038619995, -1.5043244361877441, -1.932713508605957, 1.401075839996338, 0.08956154435873032, 0.09583249688148499, -1.2203304767608643, 2.177678108215332, -0.1764339655637741, 0.10204339027404785, 1.5558655261993408, -0.12012360244989395, -1.2229292392730713, -0.3051247000694275, 0.9135539531707764, -0.40783292055130005, 0.21120794117450714, 0.8244239687919617, -0.04641665890812874, 0.9661908745765686, -2.76792573928833, 0.6632980704307556, -1.3460133075714111, 0.18644295632839203, 0.07400917261838913, -1.0356781482696533, -1.001569390296936, -0.0096300533041358, -0.4465441405773163, 0.22137965261936188, 0.6427019238471985, -1.9362517595291138, 0.21867693960666656, -0.41672608256340027, 1.3261339664459229, -0.14415417611598969, -0.43421033024787903, 0.5055971741676331, -0.6517626047134399, -1.7960433959960938, -0.13394159078598022, -0.03675342723727226, 0.3284190595149994, -1.306342363357544, 0.19455654919147491, 0.6289801001548767, -0.3166619539260864, -0.22072826325893402, 0.378892719745636, 0.4196450710296631, 0.047134771943092346, 0.7791458964347839, 0.5592765212059021, 0.27495884895324707, -0.47097986936569214, -0.02278791181743145, -1.007918119430542, 0.04819014295935631, -0.2825874090194702, -0.6388084292411804, 1.2327584028244019, -0.5484961271286011, -2.05289888381958, 1.6786421537399292, 0.36599475145339966, -0.9905301332473755, 1.0942426919937134, -0.12526856362819672, -1.2074470520019531, -1.926566243171692, 0.5193137526512146, 0.38400429487228394, -0.8246327042579651, -1.8159618377685547, 1.502182960510254, -0.37448617815971375, -1.2151362895965576, 1.2052922248840332, 1.562614917755127, -1.4602632522583008, -0.9250308871269226, -0.12466471642255783, 1.6663775444030762, 0.2687014639377594, 1.080823302268982, 0.2707424759864807, -0.08341967314481735, -1.014042854309082, -0.4856501519680023, -1.5927590131759644, -0.10668139904737473, 1.6507198810577393, 0.4351542592048645, -2.657538652420044, -0.20800606906414032, -0.28316712379455566, 0.5895197987556458, 0.9711948037147522, 0.9783328175544739, 0.26449620723724365, -1.600669503211975, -1.41777503490448, -0.8601064682006836, 0.7859649658203125, -0.2741149961948395, -0.002417516428977251, -0.015190365724265575, 2.112220525741577, 0.034844864159822464, 1.392976999282837, -0.9964472651481628, -0.37394583225250244, -1.03293776512146, 0.1326114684343338, -0.1403397023677826, 0.18586501479148865, 0.5819725394248962, -0.11790700256824493, -0.9487362504005432, -0.2201676219701767, -0.07117858529090881, 0.13566423952579498, 0.38936716318130493, 0.9897391200065613, -0.13824594020843506, 1.9498573541641235, 0.7145774364471436, 1.2092183828353882, 1.367142915725708, -0.17099304497241974, 1.1766859292984009, 0.5593791604042053, 0.5951055288314819, 0.55531245470047, -0.3579879403114319, 1.5205166339874268, 2.7576406002044678, -1.1360582113265991, -1.0844720602035522, -0.872322678565979, -0.33464792370796204, -1.1761342287063599, 1.6199402809143066, -0.646628737449646, -0.5642049312591553, -0.7098937034606934, -1.6786748170852661, 1.91614830493927, -0.7748063206672668, 0.02224140614271164, 0.39136362075805664, -0.34124770760536194, 1.1110666990280151, -0.630172610282898, -0.813854992389679, -0.16175998747348785, 0.5635581016540527, -0.2317514717578888, -0.40917906165122986, 1.4565774202346802, -0.20560328662395477, 0.5002855658531189, -0.5574907660484314, 0.10561010986566544, 0.08845045417547226, 0.7969086170196533, 0.5085499882698059, 0.1166481003165245, 1.2551348209381104, -0.29789888858795166, -0.8565331697463989, 0.894995391368866, 0.05254960060119629, 0.945690929889679, -0.22914627194404602, -0.27213019132614136, 0.5375925302505493, 0.1760753095149994, -1.3829907178878784, 0.35675346851348877, 1.1105808019638062, -1.2649949789047241, 0.7640631198883057, -0.7753177881240845, 0.2681683301925659, 1.5110571384429932, 0.538837194442749, 1.4256409406661987, -0.693760871887207, -0.4191925823688507, 0.27945926785469055, -0.5899516940116882, 0.0857163816690445, -0.8485438823699951, 0.2345154583454132, 1.0323925018310547, -0.7830655574798584, -0.16250616312026978, -1.0023667812347412, -0.5572307705879211, 0.4937707185745239, 0.34665295481681824, -0.7455806136131287, 0.27294406294822693, -0.4608429968357086, 0.7215880155563354, 0.12199120223522186, 0.28873956203460693, 1.047023892402649, -0.7345213890075684, 1.1883004903793335, 0.23298633098602295, -0.8105087876319885, 0.09632068872451782, -1.1463664770126343, -0.6836847066879272, -1.3659300804138184, 1.1821047067642212, 0.6572018265724182, 0.6162315011024475, 1.21951162815094, 0.4883577227592468, -0.2773679792881012, 0.7381356358528137, -1.0768324136734009, 0.6218093037605286, 1.9199751615524292, -1.6991162300109863, 0.7255235314369202, 1.2841384410858154, 0.882360577583313, -0.13080629706382751, -0.9048991203308105, 1.1940536499023438, 1.6445637941360474, -2.304033041000366, 0.5816665291786194, 0.1128741204738617, 0.6305757761001587, -0.8826507925987244, 0.19559264183044434, -0.4157753884792328, 0.3902537524700165, -1.1502329111099243, 0.6803563237190247, -0.6257784962654114, -0.8321322202682495, -0.7194294333457947, -1.1359689235687256, -0.42049095034599304, -0.1956702023744583, -0.1097879409790039, -1.8847658793674782e-05, 0.7522122263908386, -0.5151993036270142, -0.014981281012296677, -1.1507172584533691, -0.08322295546531677, 1.7289403676986694, 0.5921145677566528, 1.1195977926254272, -0.6675671935081482, -1.0337820053100586, -1.8678849935531616, -0.3628462255001068, 0.2669772207736969, -0.14933225512504578, 0.42545801401138306, -1.2958967685699463, -0.18758107721805573, -0.8300740718841553, -1.2558354139328003, -0.5811493396759033, 0.518841564655304, -0.48735731840133667, -1.653232216835022, 1.3112380504608154, -0.08612766861915588, 0.3253161907196045, 0.33685585856437683, 1.575508952140808, 1.5777291059494019, 0.04410398378968239, -1.7311079502105713, -1.2696641683578491, 1.3325731754302979, -0.18967311084270477, 0.3893604278564453, 1.3996573686599731, -0.1384836584329605, -0.8224433064460754, 0.45912283658981323, 0.16658245027065277, 0.5222840309143066, -1.0438746213912964, -0.47184616327285767, -0.07819206267595291, -1.6035091876983643, -0.4834633469581604, 0.06901852786540985, 0.3785818815231323, -0.7405716180801392, -0.6774740815162659, -1.1405106782913208, -1.088976263999939, 0.20949064195156097, -0.027446869760751724, -0.7372843027114868, -0.3623396158218384, 0.8998810052871704, -1.1643627882003784, -0.37198761105537415, 1.9868751764297485, 0.7287260890007019, -0.5811073780059814, 1.260299563407898, 1.1707535982131958, 0.25442224740982056, 1.5788393020629883, -0.644202709197998, 0.003900169162079692, -0.23303768038749695, 1.166935920715332, 1.6224182844161987, 1.1056313514709473, 0.8083582520484924, 0.8131541013717651, 1.144836187362671, -1.7220721244812012, -0.699157178401947, -1.2763500213623047, -0.030713019892573357, 1.0167944431304932, 0.3294677138328552, -0.538833737373352, 0.13522328436374664, -0.24056123197078705, -0.1077098697423935, 0.48967987298965454, 0.21937285363674164, -0.31504151225090027, -0.695777177810669, -1.1636712551116943, -1.782847285270691, -1.6580686569213867, 0.8849354386329651, 2.021883010864258, 0.4854070842266083, -1.0628600120544434, 0.6257258057594299, -0.5480756163597107, 1.0511871576309204, -0.16412144899368286, -2.3325231075286865, -0.2947578430175781, -1.291463851928711, -0.5391417741775513, -1.0400662422180176, -0.8870583176612854, -0.3525274097919464, -0.13091416656970978, 1.1560766696929932, 0.5213583707809448, 0.43050676584243774, 1.95644211769104, -1.1051018238067627, -0.10310366749763489, -0.21088646352291107, 0.1545064002275467, -0.3788451552391052, -0.2850664258003235, 0.8980470299720764, -1.0713621377944946, 0.46201762557029724, 1.5997458696365356, -0.43120718002319336, -1.1060490608215332, -0.25198304653167725, -0.4508764445781708, -0.4565922021865845, 0.47357815504074097, -2.075568437576294, -0.38599488139152527, 1.367519736289978, -0.21290987730026245, -0.8565711975097656, -0.6253927946090698, 0.952056884765625, 0.00054086500313133, -0.2305038869380951, 1.8989460468292236, 0.37639403343200684, 0.09601521492004395, 2.148441791534424, -1.7108824253082275, 1.18376886844635, -0.8099319338798523, 0.377922385931015, 0.1321054846048355, 1.8599202632904053, -1.4583247900009155, -0.44019997119903564, -0.5230602622032166, 0.23145778477191925, -1.2835246324539185, -0.6876919269561768, 0.2746003270149231, 0.5151455402374268, 0.9455969333648682, 1.519220232963562, -0.6651957631111145, -0.08292629569768906, -0.9190462231636047, 1.2289774417877197, 1.2593969106674194, -1.1759395599365234, 0.8411297798156738, -1.7519370317459106, -1.2453304529190063, 1.0535659790039062, -1.7107353210449219, 1.6309393644332886, -0.21223293244838715, -0.21162208914756775, -1.7290019989013672, 0.6607235670089722, -0.5365511775016785, 0.9760404825210571, 2.1862730979919434, -0.9934675097465515, -0.09264957904815674, -0.02591750957071781, 0.8537804484367371, -0.9586483240127563, 0.16009221971035004, 0.25682565569877625, -0.5797943472862244, 0.6409254670143127, 0.6086215972900391, 0.01855318434536457, -1.3406792879104614, -1.695542573928833, 1.282575011253357, 1.4993410110473633, -0.2069827914237976, 0.038893602788448334, 0.34030354022979736, 0.4526049494743347, 2.065761089324951, -0.021860603243112564, 1.8169745206832886, 0.6356187462806702, 0.08835356682538986, -2.1252057552337646, -0.3369360566139221, -1.1501296758651733, -1.0309709310531616, 0.490092009305954, -0.15069440007209778, -0.053724631667137146, -0.053120244294404984, 0.8966649770736694, 0.9085458517074585, 0.6391503214836121, -0.05306566134095192, -0.11996914446353912, 1.402032494544983, 1.8462705612182617, 1.7407886981964111, -0.19978192448616028, -0.12010478973388672, 0.7719765901565552, -1.3587673902511597, -0.4331737458705902, -1.050351619720459, -0.6715102791786194, -1.6134650707244873, -1.9393833875656128, -0.31003087759017944, -1.3385977745056152, 1.6236035823822021, -0.306275337934494, 0.37959808111190796, -0.013605082407593727, 0.8073347210884094, 0.9968653917312622, -0.3910740911960602, 2.462376594543457, -1.517433762550354, 0.06451211124658585, 0.12781062722206116, -1.0944383144378662, 0.08734413236379623, 1.3730714321136475, 0.29358410835266113, 1.8888887166976929, -1.0226290225982666, -0.1571800410747528, 0.5807765126228333, 0.8572481274604797, -1.9495688676834106, 0.2451871931552887, 1.8459659814834595, -0.6716399192810059, 1.2115724086761475, -0.7613635659217834, 1.085781455039978, -0.18004058301448822, -0.8161455988883972, 0.06976581364870071, 1.3245025873184204, 0.09064526855945587, 1.1169929504394531, 0.10519702732563019, -1.2072970867156982, 1.2168391942977905, 0.1381184160709381, -0.9417393207550049, -0.43388769030570984, -1.0460963249206543, 0.09683392941951752, 0.6101511120796204, -1.2544127702713013, 1.3736510276794434, -0.03647736459970474, -1.0256704092025757, -0.8062249422073364, -0.12280008941888809, -2.4452733993530273, -0.5831454992294312, -0.4681161344051361, 0.3397846519947052, -0.07634960860013962, 0.4682807922363281, 0.2592657208442688, 0.6473129987716675, -0.3409557640552521, 0.9730544090270996, 0.8042595982551575, 0.5818071961402893, -0.34264102578163147, -1.3129477500915527, -0.944323718547821, 1.331730842590332, 0.8009086847305298, -1.2658612728118896, -1.5387935638427734, -0.5764387249946594, 2.644263744354248, -1.2505629062652588, 1.7320111989974976, 1.7622836828231812, -0.5688554644584656, 0.6625726819038391, 1.5624361038208008, -0.10317377001047134, 1.770340919494629, -0.15209802985191345, -1.1263291835784912, 0.7558082938194275, -0.2229028046131134, 0.9099335074424744, -0.6047089099884033, 0.17194746434688568, 0.5707789659500122, 0.7948938012123108, 0.3728797137737274, 0.07290370017290115, 0.9015833735466003, 0.011802654713392258, 1.7370630502700806, 0.3437506854534149, 0.20655564963817596, 0.9962430596351624, -0.6076513528823853, 0.7457401752471924, 0.706136167049408, -1.3660566806793213, -0.3771646022796631, 1.5984220504760742, -0.7101674675941467, 1.9969675540924072, 0.4993937909603119, 0.54664546251297, 1.725688099861145, 0.009767130948603153, 0.17510740458965302, -0.24980837106704712, 1.4833018779754639, 0.11480820924043655, -0.6670132875442505, -0.33131030201911926, -0.6821736097335815, 0.795695424079895, -0.568469762802124, 0.46085482835769653, 1.5270049571990967, 1.5260441303253174, -1.8228137493133545, 0.35965588688850403, 1.0691149234771729, 1.2719308137893677, -0.9174875020980835, 0.2788007855415344, -0.9481331706047058, -0.26904577016830444, -1.5781422853469849, -1.4801074266433716, -2.0903143882751465, -1.6029067039489746, 0.06360528618097305, 1.0840904712677002, 0.3348817825317383, 1.9701980352401733, -0.4605672359466553, -0.6072531342506409, -0.10082835704088211, -1.0035021305084229, -1.3571385145187378, -0.9475412368774414, -0.7208912968635559, 0.9590843319892883, 0.2774137854576111, -0.39819619059562683, -1.951714277267456, -0.5135457515716553, -0.6975379586219788, 0.5127440094947815, -0.5860278606414795, 1.9634469747543335, 1.4857946634292603, 0.2898493707180023, -0.14105740189552307, -1.319962739944458, 0.33928653597831726, -0.3881849944591522, -1.1556059122085571, 0.8303510546684265, -0.6269297003746033, 0.2877432107925415, -1.022909164428711, -0.8040669560432434, -0.5500836372375488, 0.44129717350006104, -0.7614355087280273, -1.4599405527114868, -0.20720918476581573, 1.5162960290908813, -1.9611579179763794, -1.8931225538253784, 0.2881409227848053, 1.6552119255065918, -1.3622498512268066, -1.0692112445831299, -0.2774485945701599, 0.4581930339336395, 1.4257487058639526, 1.9538681507110596, 0.47750425338745117, -0.22823993861675262, -1.50882887840271, -0.7096756100654602, -0.5517669916152954, -1.102441668510437, 0.9150419235229492, -1.1184511184692383, -1.3067549467086792, -1.4396871328353882, -1.5026298761367798, 1.220035433769226, -1.3138455152511597, -0.6804459691047668, -2.035432815551758, 0.23069894313812256, 2.053499460220337, -0.2325311005115509, -0.8562401533126831, 0.007598617114126682, -0.012421022169291973, 0.8906503915786743, 0.4895460903644562, 1.2282202243804932, -0.6055177450180054, 0.36642593145370483, -1.0996254682540894, -0.04780277982354164, 0.11358638107776642, -1.251175045967102, 1.1628004312515259, -0.21446523070335388, -1.2203760147094727, 0.005365902092307806, 1.268620491027832, 1.2177292108535767, -0.6339212656021118, -0.16722074151039124, 0.3096141815185547, -0.44240862131118774, 1.43656325340271, -0.6785371899604797, -0.17409345507621765, -0.7393226027488708, 0.4341890513896942, 0.13826896250247955, -1.601524829864502, -2.088942289352417, 1.7793099880218506, -1.7358239889144897, 0.03413262218236923, -0.5673349499702454, 1.1860222816467285, -0.8004851937294006, 0.036429513245821, -0.42791077494621277, -0.6076475977897644, -1.043353796005249, 0.2233130782842636, -0.2796478867530823, -1.1576197147369385, -0.002996626775711775, 1.200842022895813, 0.8196439743041992, 1.9474502801895142, 0.7654650211334229, 0.5446376204490662, 0.3695266544818878, -1.3905506134033203, -0.9259217977523804, -0.36526891589164734, -0.8099931478500366, 0.5709061622619629, 2.9508419036865234, 1.0362378358840942, -1.952682375907898, 1.2260833978652954, -0.5519667863845825, 0.5475103259086609, -0.02583698183298111, -1.1397159099578857, 0.6001969575881958, 0.30283913016319275, 0.6528304219245911, -0.1274232566356659, 0.4258941113948822, 1.617605447769165, -1.3455005884170532, -0.5013144612312317, 0.5863159894943237, 0.2018042355775833, 0.15190844237804413, 0.27327558398246765, -0.33320096135139465, 1.3833839893341064, -0.8223105669021606, -0.7521100044250488, 0.07842801511287689, -0.8201906681060791, 1.7247556447982788, -2.6614279747009277, -0.45080021023750305, -0.5662928819656372, -1.2189197540283203, 0.16608090698719025, 0.698689877986908, -0.8623906970024109, 0.23345129191875458, -0.09261882305145264, 0.4325021505355835, 0.09539426863193512, 0.9349466562271118, 1.3809643983840942, 2.09279203414917, -0.918776273727417, -0.21758460998535156, -0.11312739551067352, 1.7770720720291138, 0.23042979836463928, 0.8505323529243469, 0.5109968185424805, -1.6496422290802002, 0.32377415895462036, -0.602832019329071, -0.4701641798019409, 0.20873622596263885, -0.5225575566291809, 0.9779903888702393, 1.0731449127197266, -0.9029017090797424, 2.03711199760437, -0.5222962498664856, 1.128764033317566, 1.1599047183990479, -1.361448049545288, -0.5509979724884033, -0.7037550210952759, -0.27040886878967285, 0.38493114709854126, 0.8606027364730835, 1.407552719116211, -0.235494926571846, -1.4898381233215332, 1.4545481204986572, 0.6346235275268555, -0.980434775352478, 0.2577213943004608, 0.11710771173238754, 1.1288820505142212, 1.13687264919281, -0.4350360333919525, 0.8706746101379395, 1.4986188411712646, 0.8090082406997681, 1.4270555973052979, 0.8696872591972351, -2.0712764263153076, 0.014630179852247238, 0.09263864904642105, 0.23902851343154907, 0.7784809470176697, 0.46001046895980835, 0.5942183136940002, 0.8282901048660278, 1.0604274272918701, 0.7019805312156677, -1.0061089992523193, 1.2126954793930054, -0.3031121790409088, -0.009906894527375698, -0.6362451314926147, -0.6497741937637329, -0.3119172751903534, -1.3938175439834595, -0.3714264929294586, 0.4878941476345062, -0.4189971387386322, -0.4054747223854065, -0.5160502195358276, 0.8502368927001953, 0.8185058236122131, -0.28524285554885864, 0.5672953724861145, -0.07431857287883759, 1.4420433044433594, 0.7488492727279663, -1.0394952297210693, 0.6826720833778381, -0.7174988389015198, 0.06573627144098282, -1.8482414484024048, 0.39364686608314514, -1.1832014322280884, -0.30454590916633606, -1.369057059288025, 0.20582430064678192, -0.6992347836494446, -0.03164517506957054, 0.5748726725578308, 0.1581452190876007, 2.274757146835327, -1.4729982614517212, -1.4609726667404175, 0.157956063747406, -0.3285786807537079, 1.4473819732666016, 0.3328859508037567, -0.28139349818229675, 0.3431456685066223, -0.33785519003868103, -1.5849294662475586, 0.6497657895088196, -1.5237715244293213, 0.7575492262840271, -1.773332953453064, -0.7938652038574219, -0.12424622476100922, -0.43909916281700134, -0.11141885817050934, -1.0948195457458496, -2.1183571815490723, 1.9290878772735596, 0.4156012535095215, -0.6701850891113281, 0.7804708480834961, -0.38519978523254395, -0.026102876290678978, -0.4684341847896576, 1.4924181699752808, -1.1997127532958984, 0.7096508145332336, -1.6338328123092651, 0.16436296701431274, 1.3096344470977783, -1.1168991327285767, -0.19394877552986145, -0.7397366166114807, 0.3879784345626831, -0.6082841157913208, 0.6500323414802551, -1.19076669216156, -0.04134676977992058, 0.9198186993598938, -1.1042990684509277, -0.6252121925354004, -0.4117490351200104, 1.2402232885360718, 0.12861095368862152, -0.6580348014831543, 0.011053265072405338, 0.484988272190094, 1.6994727849960327, 1.8697279691696167, 2.225825309753418, 0.6837860941886902, -0.0565946102142334, -1.5788110494613647, 1.031195878982544, -0.9408618807792664, -0.6607587933540344, -0.6545715928077698, -0.06807595491409302, -0.13005387783050537, -0.38653433322906494, -1.3919336795806885, -0.5101706981658936, 1.1246416568756104, 0.48296236991882324, -1.9195791482925415, -0.17481786012649536, 0.08379548043012619, 0.7469152808189392, 1.1028058528900146, 0.70392906665802, 1.5564906597137451, -0.3710263967514038, -0.5028008818626404, -1.9776606559753418, 1.9539527893066406, 1.7049264907836914, -1.371362328529358, -2.001777410507202, 0.008674458600580692, -0.4095555543899536, 0.1416737288236618, -0.7138634324073792, 0.555303692817688, -0.8157246708869934, 1.3752593994140625, 1.144327163696289, 1.6667628288269043, -0.3976075053215027, -0.2288840413093567, 0.11803209781646729, 0.6445693373680115, 0.09291724860668182, 0.38619574904441833, 0.78011554479599, -1.702319622039795, 1.2438234090805054, 1.5449284315109253, -1.8007569313049316, -0.9866524934768677, 0.44015833735466003, 1.2856230735778809, 0.944665253162384, 0.15235044062137604, -0.1718175858259201, 0.35517680644989014, -1.4756196737289429, -0.3548455238342285, 1.5778930187225342, -1.6369128227233887, 0.297842800617218, -0.4504188299179077, -2.5130722522735596, -0.14601318538188934, 0.6640787720680237, 0.6960991024971008, 0.3035120964050293, -0.22713439166545868, -0.8359264135360718, 1.0855857133865356, -1.0930821895599365, -0.48338523507118225, -0.26877647638320923, -0.770635724067688, 0.6678245663642883, -0.4426763355731964, -1.5576434135437012, -1.5130022764205933, 1.9126007556915283, -0.8947705626487732, 1.3990892171859741, -1.3006188869476318, 1.6131747961044312, -1.2965707778930664, -0.5676567554473877, 0.5756366848945618, -0.1114133968949318, -1.616140365600586, 0.016048384830355644, 1.3730227947235107, 0.01554662361741066, -0.07028191536664963, 1.5815261602401733, -0.18175435066223145, -0.04192911833524704, 1.4327986240386963, -0.8557531833648682, -0.4155113697052002, 1.1792054176330566, -1.0917755365371704, 0.7660345435142517, 0.5893987417221069, 0.29778745770454407, 0.019808895885944366, 1.0066726207733154, 0.5727275609970093, -0.942712128162384, 1.2287009954452515, -0.4620932638645172, 0.572173535823822, -0.400477796792984, -0.07784535735845566, 0.7883405685424805, 0.24653983116149902, 0.011794650927186012, 0.13525643944740295, 1.0117590427398682, 0.2907240390777588, 0.4119253158569336, 1.85769784450531, -0.3187984824180603, -0.293634831905365, 0.7314751744270325, 0.18783092498779297, -1.8684402704238892, 0.488789826631546, 1.3632808923721313, 1.4905015230178833, -0.01728592813014984, -0.722743034362793, 0.18850260972976685, 1.1837748289108276, -1.2165513038635254, -2.7278685569763184, -0.9807906746864319, -0.058676015585660934, 0.9496213793754578, 1.0021533966064453, -0.9562498927116394, 0.06937728822231293, 0.778374195098877, 0.3465784788131714, 0.25385233759880066, -1.120514988899231, 0.46711117029190063, -0.9923709630966187, -0.07924901694059372, 0.39105239510536194, -0.9749622344970703, 1.3741759061813354, 0.22588957846164703, -0.44825661182403564, 2.4333226680755615, -0.5386723279953003, -0.6273688673973083, -0.08833429217338562, -1.7775169610977173, -0.5504086017608643, 0.4612070620059967, 0.6592003107070923, 0.6891653537750244, -0.9155827760696411, -1.2657850980758667, 0.7197169065475464, 1.4551395177841187, 2.5006134510040283, -0.5578492283821106, 0.515180766582489, -0.18961158394813538, -2.0050153732299805, 0.7296176552772522, -0.7766984105110168, -0.6934176087379456, 1.024896264076233, 0.9710820913314819, -1.149901270866394, 0.24654437601566315, 0.5060843229293823, -1.4357357025146484, 1.6113543510437012, 0.5732501149177551, 0.9241198301315308, -0.2234753519296646, -0.501043975353241, 1.076195240020752, 1.2562098503112793, -1.6815763711929321, -0.6275820732116699, -0.37283629179000854, 0.32536256313323975, -1.0124149322509766, 0.6408810615539551, 3.0660934448242188, 0.9749757051467896, 0.9100260138511658, -0.3025703728199005, 0.20921112596988678, -1.3488520383834839, 0.41709065437316895, 1.9329679012298584, 0.601068377494812, 0.9022808074951172, 0.6199178695678711, -0.3820343017578125, 0.7101209759712219, -0.06209303438663483, -0.15103581547737122, -1.1063623428344727, 1.7842341661453247, -0.9364523887634277, -0.947590172290802, 1.5708860158920288, -0.005034332629293203, -0.398146390914917, -0.11900164932012558, 1.297466516494751, -0.7446805834770203, 2.7681026458740234, 0.5784987211227417, -0.5089004635810852, 1.0628960132598877, -0.5009397268295288, 2.2046144008636475, 0.2855919897556305, -0.783189594745636, 0.17319531738758087, 0.7476520538330078, 0.2689528167247772, -0.7012209892272949, 0.12371517717838287, 1.107806921005249, 1.1098960638046265, -0.5905064940452576, 0.7057124376296997, -0.35127905011177063, 1.285496711730957, -2.090097188949585, -0.8398289680480957, -0.4382416307926178, -0.7832157611846924, -0.16439957916736603, 0.978387713432312, 0.058271732181310654, -0.7329452037811279, -0.15417972207069397, -0.3899422287940979, 0.5773377418518066, -0.47150135040283203, -0.1465306431055069, 0.7212669253349304, 1.4143487215042114, 0.255311518907547, -0.3755844235420227, -0.8151174783706665, 1.2598795890808105, -0.7516022324562073, 0.07099822163581848, 1.4823211431503296, 0.14038823544979095, 0.23437610268592834, 0.799150824546814, 1.1456955671310425, 0.8939585089683533, 0.6616023778915405, 2.2066664695739746, 1.3490301370620728, 0.3867020606994629, 0.3105432689189911, 0.40117979049682617, 0.9697721004486084, -1.8041547536849976, -0.3951702415943146, 0.10642405599355698, 1.440949559211731, -0.12701515853405, -0.04978662729263306, -1.220970869064331, 1.236770749092102, 0.9125277400016785, -1.2819067239761353, 0.45769429206848145, 0.8254850506782532, -0.45648062229156494, 1.9202885627746582, 0.07346225529909134, -0.12817391753196716, 0.14331801235675812, -1.1381115913391113, -0.3594534695148468, 1.1201002597808838, 0.09660819917917252, 0.7924465537071228, 1.403077244758606, 0.6110184192657471, 0.20294071733951569, -1.7066363096237183, 1.2779598236083984, -0.7961759567260742, -0.22141461074352264, 0.8103823065757751, 0.22135034203529358, -0.4602406322956085, 0.6505894064903259, -0.2194090187549591, -1.2589640617370605, -0.0420999713242054, 0.0072114416398108006, 0.029417430981993675, 0.27778223156929016, -0.5873515605926514, -0.3711370527744293, 0.41204172372817993, 0.6901347637176514, 1.2013208866119385, -1.1783298254013062, 0.030512461438775063, 0.12873829901218414, 0.6438804268836975, -1.3622636795043945, 0.8044654130935669, -0.5357439517974854, -0.021636594086885452, -2.394243001937866, 0.8192092776298523, -0.2543962597846985, 0.6230857968330383, 2.198376178741455, -1.6053080558776855, 0.34672731161117554, -1.4879631996154785, 0.14455389976501465, -0.4637918174266815, -0.8812130093574524, -0.8591528534889221, -0.39091262221336365, -1.8843905925750732, -0.6757696866989136, 0.44519829750061035, -1.1526215076446533, -1.1890485286712646, 0.4453412890434265, -0.785824179649353, 0.6698769927024841, -0.48433682322502136, -0.8164300322532654, 0.14338034391403198, -0.6993908882141113, -0.060681331902742386, 1.0301449298858643, 2.3186709880828857, 0.6136144995689392, 0.03349016234278679, 1.005646824836731, 0.3185383081436157, -1.1509554386138916, 1.2094217538833618, 0.5452023148536682, -0.0754772499203682, 0.4158807098865509, -1.1453330516815186, 1.4134200811386108, 0.32452136278152466, 0.5322546362876892, 0.8541932106018066, 0.046943530440330505, 1.0816222429275513, -0.20289112627506256, 1.1223957538604736, 0.9604119658470154, 1.7755999565124512, -1.4004791975021362, -0.30649346113204956, 1.8171411752700806, 0.48002657294273376, -2.636648416519165, 0.8009931445121765, -1.1119308471679688, 1.5634816884994507, -0.6554456949234009, 3.170358657836914, -0.012850173749029636, -0.34243160486221313, -0.308313250541687, -0.1413663923740387, -0.58232581615448, -0.6539773941040039, 0.15536247193813324, -0.08448400348424911, -1.9143850803375244, -0.5112048983573914, -0.7633864879608154, 0.4586748778820038, 0.8668140172958374, -0.021131055429577827, 1.9240049123764038, 0.18785445392131805, 0.8891956806182861, 0.3010774850845337, -1.7737170457839966, -0.6174716949462891, 0.3825080990791321, -1.4484782218933105, 1.1029549837112427, 0.7991090416908264, -0.2731205224990845, 1.215352177619934, -0.5471682548522949, -1.2995353937149048, -1.1170809268951416, -1.0404924154281616, -0.6626980304718018, 0.5902515053749084, 2.0097250938415527, -2.537773609161377, -1.5134410858154297, 1.044541358947754, 0.19865810871124268, 2.240687847137451, -0.4800567030906677, 1.364704966545105, -0.04420074447989464, 0.35592979192733765, -0.013396037742495537, 0.009250680916011333, 2.8977267742156982, 1.2667644023895264, -1.138597846031189, -0.6712172031402588, 0.3691169321537018, 1.9641095399856567, -0.22607846558094025, -1.4867675304412842, -0.9325985908508301, -0.31512507796287537, 0.7862627506256104, 1.4519999027252197, -1.5009101629257202, 0.5691066980361938, -0.18843109905719757, -0.08726540952920914, -0.12003482133150101, 0.2253178060054779, 0.5823936462402344, 0.14328637719154358, 1.1808323860168457, 0.6994128823280334, -0.9115381240844727, 1.6897642612457275, 0.6329337358474731, 1.9117716550827026, -1.7654188871383667, 0.6237030029296875, -0.6442907452583313, -0.07409632951021194, -0.02628679759800434, 2.2479653358459473, -0.052924204617738724, 2.3036468029022217, -2.7922935485839844, 0.7040112614631653, 0.15179140865802765, 0.4308401644229889, 0.8432902693748474, -1.4591153860092163, -0.05999533087015152, -0.6048274636268616, -0.19314232468605042, -0.4473264217376709, -1.5251798629760742, -2.2818658351898193, 0.16929131746292114, -0.8482480645179749, -0.1689348667860031, -0.013784854672849178, -1.1897281408309937, 0.755558431148529, 0.7737674117088318, 0.6028295755386353, 0.5386959910392761, 1.0689057111740112, -1.0165852308273315, 0.7575514316558838, 0.8569669723510742, 1.364456295967102, 0.5580812692642212, -0.5246194005012512, 0.2162373661994934, -0.5719372034072876, 0.8429656028747559, -0.8933548927307129, -0.20155148208141327, -0.008696100674569607, -0.9113970994949341, -1.5454144477844238, -0.10963943600654602, 0.15991926193237305, -0.45983627438545227, 1.2932020425796509, 1.388258695602417, 1.2762361764907837, 0.30566829442977905, 1.1277607679367065, 1.0399926900863647, -0.40468353033065796, 0.6607378721237183, 1.1385879516601562, -0.3541634976863861, -1.2301175594329834, 0.20244160294532776, 1.1494613885879517, -0.05630558729171753, 1.1646407842636108, 0.4665622413158417, -0.28833308815956116, 0.15245412290096283, 0.2125004380941391, 0.2789161205291748, -0.4958971440792084, -0.8582355976104736, 0.8975745439529419, 0.2687954008579254, -0.3351154625415802, -1.4865142107009888, -0.6390364170074463, -0.05597148463129997, -1.0213780403137207, -0.7962063550949097, -0.3707766532897949, -2.1846961975097656, -1.1506375074386597, 0.15747973322868347, -2.24989652633667, -1.4508756399154663, -1.7153595685958862, 0.9316906929016113, -1.3125505447387695, 0.08363451808691025, -0.7852234244346619, -0.12532494962215424, -0.6026338934898376, 0.4808080196380615, -1.2446866035461426, 1.7346446514129639, 0.6283039450645447, 0.1257830560207367, -0.7082250714302063, 1.4472033977508545, -2.217215061187744, -0.08877421915531158, 0.8657022714614868, -0.17969952523708344, -0.6690601110458374, -1.9835842847824097, 0.7537633776664734, 0.6135523319244385, -0.9637092351913452, 1.8901110887527466, 0.4577060639858246, -0.6325873732566833, -0.11249957233667374, -0.10718300193548203, 0.5966426134109497, -0.41214582324028015, 1.0102051496505737, -0.9211956262588501, 2.14420223236084, -0.5725297927856445, 0.9634842276573181, 1.804824709892273, 1.1220396757125854, 0.36298006772994995, 1.2991886138916016, -0.5157289505004883, -1.2731366157531738, 0.8656793832778931, -0.15967102348804474, 0.06349142640829086, 0.4725593328475952, -0.6688099503517151, -0.15062178671360016, -0.036581214517354965, -0.11073154956102371, 0.48173707723617554, -0.04673825949430466, 0.443111777305603, -0.884548008441925, 0.22236572206020355, -1.2192296981811523, -0.5297422409057617, -0.05821976065635681, 0.08944641053676605, -0.8722453117370605, 0.924811065196991, -0.16924208402633667, 0.8790608048439026, -0.6739726066589355, 0.24497073888778687, -0.5468860268592834, 1.7259231805801392, -0.20894283056259155, -1.1713776588439941, -0.32050400972366333, -0.19086329638957977, -0.1688501238822937, -0.20213273167610168, -0.5419114232063293, 0.7450330257415771, 0.5812878012657166, -0.7923881411552429, -0.5042446255683899, -2.402533769607544, -2.1367101669311523, 1.5841811895370483, 1.4391437768936157, 0.6738386750221252, -2.216965436935425, -1.0472190380096436, -0.9795429110527039, -0.9942536354064941, -0.6532489061355591, -0.5573933124542236, -0.034047726541757584, 0.07856069505214691, 1.4172906875610352, -1.1599539518356323, -0.5968075394630432, -0.1988578587770462, -0.37237581610679626, -0.33685481548309326, 0.1501404196023941, 1.7390440702438354, 0.5784443020820618, -0.8755105137825012, -1.299402117729187, -0.44066452980041504, -0.46309059858322144, 0.7141185998916626, -0.06006557494401932, -1.8916666507720947, -0.7141631245613098, 0.24806445837020874, 0.7444069981575012, 0.11721885949373245, 0.023360829800367355, 0.17824235558509827, 0.7058266401290894, 0.6814275979995728, 1.2391935586929321, 0.3657185733318329, -0.1836436241865158, 1.455128788948059, -0.03803861513733864, -2.072028875350952, -1.3995270729064941, 0.7282345294952393, -0.29571783542633057, 1.1909948587417603, -0.26120254397392273, 1.1723535060882568, -1.706228256225586, -0.09663903713226318, 1.7072346210479736, -0.7548621296882629, 0.12017618864774704, -1.107161283493042, -1.0866940021514893, 1.5107804536819458, 0.3993813693523407, -1.2509846687316895, -0.8118950128555298, 0.07237987220287323, -0.46019527316093445, -1.0274741649627686, -1.5772329568862915, -0.6881378293037415, -1.1817283630371094, 1.1779499053955078, 0.16199201345443726, 1.4109901189804077, 0.7502924203872681, -0.9717003703117371, -1.1462182998657227, 0.7081499099731445, 0.26258864998817444, 0.7518513798713684, 0.562879204750061, 0.8379868865013123, -0.637874186038971, 1.3575481176376343, 1.178561806678772, 0.33386021852493286, 0.8826302886009216, 0.9661771059036255, -1.295868992805481, -1.494533896446228, -0.6307963132858276, 0.3568885326385498, 0.7243652939796448, 0.28014346957206726, 1.6049903631210327, 0.6174285411834717, 1.6338000297546387, 0.9704845547676086, 0.9906306266784668, 0.5025238394737244, -1.3700271844863892, -0.6303232908248901, 0.5372734665870667, -1.8844937086105347, -0.8992665410041809, -0.5123537182807922, -0.8508500456809998, -0.01095679122954607, -0.5935937166213989, -0.06549147516489029, -0.7755470871925354, 0.3513989746570587, -2.292370080947876, -0.39998292922973633, -0.012069791555404663, 2.005554676055908, -1.745413064956665, 0.5739617347717285, -0.6806972026824951, -1.2174137830734253, 2.014785051345825, 0.3554113507270813, 1.6282957792282104, -1.1886569261550903, 0.05738583207130432, -0.5697619318962097, -0.9893890619277954, -2.234889268875122, 0.3329828977584839, 1.013073205947876, 2.2861928939819336, -0.6591397523880005, 1.1256202459335327, 1.3309794664382935, 1.868175983428955, -1.0505543947219849, 0.2248971164226532, 1.0921094417572021, 1.6011126041412354, -0.7943795323371887, 0.7598776817321777, -1.6285964250564575, -0.811477780342102, -0.12886157631874084, -1.64358389377594, 0.11632313579320908, -0.0897437334060669, 3.418867349624634, 0.5307906866073608, -0.5895872712135315, 0.13324901461601257, -1.3059558868408203, -0.35309624671936035, -0.5014845728874207, -1.364129662513733, 0.19517377018928528, 0.7197655439376831, 0.6224006414413452, -0.655117928981781, -1.2377692461013794, -0.9342034459114075, 1.5739405155181885, 1.0027137994766235, 0.4685201942920685, 0.8442180156707764, 1.1214873790740967, -0.45233142375946045, 0.7179957032203674, 0.0338750034570694, -0.12613680958747864, 1.5437442064285278, -0.7984792590141296, 0.4368647336959839, 1.0311973094940186, 0.7629926800727844, -1.5814357995986938, -0.05910423770546913, -0.9987607598304749, 0.16605103015899658, -0.10500086843967438, -1.076140284538269, -0.15291756391525269, 0.4180000424385071, -0.7301961779594421, 0.17510156333446503, 0.19853928685188293, -0.11831191182136536, -2.49780011177063, -1.5641489028930664, -0.43049776554107666, 0.7165066599845886, 0.5299323201179504, -0.5320138335227966, -0.09168318659067154, -0.644870936870575, -1.2379891872406006, 2.8509409427642822, 0.8401954174041748, -0.5807979106903076, -1.0398653745651245, -0.20239710807800293, 1.7179726362228394, 0.0792464166879654, -1.3093981742858887, -1.8522067070007324, 0.731664776802063, -1.2668787240982056, -0.749254584312439, -0.22995513677597046, 0.16210436820983887, 0.28881898522377014, -0.10170324146747589, -0.4074162244796753, 0.9666532278060913, 0.9985283017158508, -0.4806390702724457, -0.44220101833343506, -0.5394516587257385, -1.1819158792495728, 0.904075026512146, 0.02686404436826706, 2.1989309787750244, -0.5297955274581909, -0.6452092528343201, -3.2366573810577393, 0.29212453961372375, 0.3134971559047699, 1.3062410354614258, 1.3507293462753296, 1.365992784500122, -0.1559128314256668, 1.3588099479675293, -1.0862915515899658, -1.9248119592666626, -0.6281979084014893, -0.2721918523311615, 0.8312883377075195, -2.202514886856079, -1.40865957736969, -1.9125049114227295, -2.4101011753082275, 0.8041374683380127, -0.27812936902046204, 1.538588285446167, -0.34235838055610657, -0.7356865406036377, -0.219509556889534, -0.4583311378955841, -1.3131519556045532, -0.9235681891441345, 0.3249853551387787, -0.7521902918815613, 0.33384501934051514, -1.8569765090942383, -1.3245950937271118, 1.334373116493225, -0.4930991530418396, 0.30593565106391907, -1.3450472354888916, -0.6464681625366211, 1.6545969247817993, -0.045597273856401443, -1.293116569519043, -0.1527729034423828, 0.9163162112236023, -0.07739990204572678, -0.07948268949985504, -0.5218825936317444, 0.39753100275993347, -0.2900192439556122, 0.9754095673561096, -0.8410808444023132, -0.2541152834892273, -1.2150113582611084, 0.8812957406044006, 1.1155495643615723, 0.15097284317016602, -1.1447196006774902, 0.7874465584754944, 0.8756666779518127, -0.8879587054252625, 1.2509076595306396, -0.1155766174197197, 0.7487856149673462, 0.4544467628002167, -0.5821667313575745, 1.7288511991500854, -0.3997000753879547, -0.07843223959207535, -0.6266594529151917, 0.6480123996734619, 1.4373071193695068, -0.07583258301019669, -0.4411911070346832, 0.7189557552337646, -0.6912691593170166, -1.5275059938430786, -1.6702115535736084, 1.3042271137237549, -0.216547891497612, 0.6034144759178162, 0.2849389314651489, 0.7553295493125916, 0.35431140661239624, -1.9592576026916504, 1.256406307220459, 1.1732667684555054, -0.5699992179870605, -0.1400369107723236, -1.0920262336730957, -0.6652234792709351, -1.004878044128418, -0.8556952476501465, 1.2806854248046875, 0.15423928201198578, -0.5257905125617981, 0.6365174651145935, -1.5387253761291504, 1.610068678855896, -1.1012771129608154, 0.3050972819328308, 0.4010998010635376, -1.452995777130127, -1.679525375366211, -0.25301724672317505, 0.16666333377361298, 1.2013417482376099, -0.44711562991142273, 0.2932499051094055, -2.9586293697357178, 1.1475834846496582, 1.7550429105758667, 0.3126107156276703, -0.10216906666755676, -0.3837672770023346, 1.6997102499008179, -0.5286605954170227, 0.09682050347328186, -0.14805454015731812, 1.8887243270874023, 0.8569326400756836, 2.567192554473877, -0.35298776626586914, -0.8064135909080505, 0.26760461926460266, 0.4162467122077942, 0.10571498423814774, 0.4381888210773468, 0.9437010884284973, -0.17685841023921967, 0.841046154499054, 1.369953989982605, -0.5748928189277649, -0.3930304944515228, -1.2106186151504517, -0.6856237053871155, -0.3525141477584839, -0.3785863220691681, -2.8409101963043213, 0.03606542572379112, 0.7831980586051941, -1.5949556827545166, -0.03378269076347351, 0.33504340052604675, 0.6422979235649109, 2.4862146377563477, -0.5772619247436523, -2.209322929382324, 0.5854365825653076, 0.8177801370620728, 0.8457033038139343, 0.3696243166923523, -0.7575794458389282, 0.6526201367378235, -0.17281474173069, 0.37717509269714355, -0.2884722054004669, 0.6094555854797363, 0.3300538957118988, -0.057503025978803635, 1.000659704208374, -0.5825062990188599, -0.39625364542007446, -1.1920115947723389, 1.1670703887939453, -0.9380910396575928, -0.4320567846298218, -1.1368976831436157, 1.2642267942428589, 1.812796711921692, 0.6937856078147888, -1.7117059230804443, -0.6269944906234741, -0.9384108185768127, -0.8389921188354492, -0.7915489077568054, -0.07168322056531906, 0.6010386347770691, 1.054002285003662, 0.13262401521205902, -0.40483149886131287, 0.867018461227417, -0.4530477225780487, 0.5198054313659668, 1.1012991666793823, -0.16204683482646942, 0.6195911169052124, 0.9811193943023682, 0.3822384476661682, -0.9578600525856018, -2.043658494949341, -1.0088589191436768, 1.1769567728042603, 0.554171621799469, 0.02794041857123375, -0.7249773144721985, 1.618634581565857, 0.213203564286232, -0.06469760090112686, 0.064895398914814, -0.5059561729431152, 0.5402395129203796, 1.0902211666107178, 0.6467969417572021, 3.2347676753997803, 0.10097939521074295, -0.5685110688209534], [-0.581704318523407, -2.5370030403137207, -1.5925301313400269, 0.5000801682472229, 0.5926878452301025, 0.7373540997505188, -0.048814497888088226, -0.18185384571552277, -0.06144654005765915, 0.4980085790157318, 0.06920760869979858, -0.9959399700164795, 0.908479630947113, -1.3809717893600464, -0.003219794249162078, 0.7266541123390198, 0.5716092586517334, -0.5123236179351807, -0.695022702217102, 0.32239389419555664, 1.0421830415725708, -0.8445259928703308, -0.4872085452079773, -0.028663400560617447, -2.3460090160369873, 0.24109326303005219, -0.8455718159675598, -1.2022699117660522, 0.6625321507453918, -0.6797380447387695, -0.4189428389072418, 0.43247276544570923, -1.251227617263794, 0.0454392172396183, 0.18856509029865265, -2.0945749282836914, -0.3275059461593628, 0.1574830263853073, 0.5554036498069763, 0.7566757202148438, -1.1703824996948242, 0.8519372344017029, 0.7042398452758789, -1.220300555229187, -1.4575515985488892, -0.7086344361305237, -1.0356836318969727, -0.7084448933601379, -1.6956404447555542, 0.2941124737262726, -0.7284157872200012, -1.2119990587234497, -0.17898190021514893, 1.2783122062683105, -1.1718840599060059, -0.4088270366191864, -0.9844675660133362, 1.08974027633667, 0.26644840836524963, 0.5961057543754578, -0.5821294784545898, 0.6953625679016113, -1.0856682062149048, 0.21411406993865967, 0.24390354752540588, 1.3726747035980225, -2.2084686756134033, -0.10453096032142639, -0.008452324196696281, 0.4332455098628998, 0.5958840250968933, 0.5882370471954346, 1.9522955417633057, 2.1576671600341797, 0.12314258515834808, 0.18492956459522247, -0.45621055364608765, -1.0885443687438965, 0.9773060083389282, -1.715419888496399, 0.09427116811275482, 1.5882576704025269, -2.933912754058838, -1.7310564517974854, -0.6698240637779236, -0.7965385317802429, -0.8261508345603943, 0.9118282198905945, 1.1765038967132568, -0.6745962500572205, 1.5346368551254272, -0.9218018054962158, -0.0018083787290379405, -0.9865249395370483, -0.4796026349067688, 0.9525235295295715, -2.0716745853424072, 0.21232637763023376, -0.7713364362716675, 0.34362268447875977, -0.020095838233828545, -1.146616816520691, 1.017203450202942, 0.3330266773700714, -1.416617512702942, 1.0366415977478027, 0.13029997050762177, -0.572458803653717, -0.5703038573265076, 1.7760145664215088, 0.5768116116523743, -0.1645597517490387, -0.31564265489578247, -0.4234880208969116, 0.5131523609161377, -0.24447514116764069, 0.2095211148262024, -0.07042686641216278, -0.5600208044052124, -0.17659562826156616, -1.3372284173965454, -0.47607049345970154, -2.1354451179504395, 0.5354047417640686, 0.4470503628253937, 1.7702051401138306, 1.8847615718841553, -0.2864874601364136, -0.6810681223869324, 0.3133791387081146, 2.4444806575775146, 2.5663087368011475, -0.1900770217180252, 0.39860811829566956, 1.3625010251998901, 0.49625149369239807, 0.3761809468269348, -1.2227330207824707, -0.668707549571991, -0.715508222579956, -2.276226043701172, -0.890288770198822, 1.413443922996521, 0.5112141966819763, 0.12013654410839081, -0.18378452956676483, 0.7076789140701294, -1.0124225616455078, -1.0088268518447876, -0.05559222772717476, -0.21292759478092194, 0.17349930107593536, -0.6745015382766724, -1.1592841148376465, -0.30630213022232056, 0.6828243732452393, -0.7874455451965332, -0.12511570751667023, 0.665567934513092, -1.857981562614441, -0.3804076910018921, -0.5200706720352173, -0.892837405204773, 0.1886368989944458, 0.1576053649187088, 0.5575327277183533, -1.4454563856124878, 0.2984166741371155, -0.29490774869918823, -0.4886765480041504, 1.5724050998687744, -0.3316732347011566, -1.1220911741256714, 0.44209808111190796, -1.7727439403533936, 1.2163804769515991, 1.3346508741378784, 1.6497725248336792, 0.4547122120857239, 0.6659643650054932, 0.4823414981365204, -0.9474956393241882, 0.6819251775741577, -0.6540085673332214, -1.0744564533233643, -0.7917590141296387, -0.04394149035215378, -0.3747885823249817, -0.47683972120285034, 1.2652732133865356, 0.49836575984954834, -0.6664683222770691, 0.9591662287712097, 1.7512574195861816, -0.8902958035469055, -0.27126142382621765, 1.8904215097427368, 0.31628596782684326, -0.8186054825782776, 1.2145962715148926, 0.15842436254024506, -0.38523197174072266, 0.5265290141105652, 0.5437502264976501, -1.2845450639724731, -0.1186872348189354, -0.7995718121528625, 0.862464189529419, -1.3515359163284302, -0.008022740483283997, -1.5356019735336304, 0.1966714709997177, 1.12644362449646, 0.2710370421409607, 0.023695312440395355, -0.03182927891612053, 1.5209566354751587, -1.2905632257461548, 0.9746221303939819, -0.5201618671417236, -0.23239374160766602, -1.1434303522109985, 0.3254542946815491, 0.4417528212070465, -0.5168659090995789, -0.05221053212881088, 0.014778614044189453, 1.835220217704773, 0.4168926477432251, 2.4982244968414307, 0.3970041573047638, 2.130648612976074, 0.4311380684375763, -0.019686536863446236, 0.0029975224751979113, -2.093186140060425, 0.43663033843040466, 0.27327585220336914, 1.1321626901626587, 0.5582756400108337, 2.189486265182495, 0.36466220021247864, 0.32713583111763, 0.5983303785324097, -0.7084903717041016, -0.17558275163173676, 0.34179550409317017, 0.12333495169878006, -0.29451867938041687, 0.46844586730003357, 0.7277430295944214, -1.7063729763031006, 0.07934869080781937, 0.9583417177200317, 0.1625927984714508, 0.19367213547229767, -1.6776432991027832, 1.048619031906128, -0.7510924935340881, 1.264218807220459, 0.9959505200386047, 1.1141990423202515, -1.0325032472610474, 0.6766381859779358, 0.889565110206604, -0.2459927350282669, -0.8596958518028259, -1.3322502374649048, 0.9234408140182495, 0.1246786117553711, -0.6225426197052002, -0.6712244153022766, 0.30474036931991577, -0.7676846385002136, -0.37073370814323425, -0.8861455917358398, 0.1382918804883957, -0.018679922446608543, 1.9880740642547607, 1.0311617851257324, -0.39286965131759644, -0.2738490402698517, -3.2302238941192627, -1.3464624881744385, 1.1266484260559082, -0.4006870985031128, 0.30532214045524597, 0.5573909878730774, -1.0327832698822021, -0.7175116539001465, 1.0583456754684448, -0.27654287219047546, 1.0340327024459839, -0.6352414488792419, -0.268244206905365, -0.21667474508285522, 0.5173327922821045, -0.33511167764663696, 0.17224712669849396, -0.026660248637199402, 0.10351892560720444, 0.34644490480422974, -0.2524953782558441, -0.7565988302230835, 1.0336412191390991, -0.13602736592292786, 0.9371371865272522, 1.144960641860962, 0.9839890599250793, -0.4470463991165161, -0.46857336163520813, -0.922741711139679, 0.7020469903945923, 0.7501641511917114, 1.5619920492172241, -1.0076149702072144, -2.572181224822998, 0.14713290333747864, 0.4577338397502899, -0.8296087980270386, -0.6479394435882568, 0.03552183136343956, -1.34996497631073, 0.40816229581832886, -0.5264788866043091, 1.7202492952346802, 0.8263607621192932, 1.161925196647644, -0.19456695020198822, 1.1003096103668213, 0.14214861392974854, 2.3772897720336914, -0.26369884610176086, -0.23621460795402527, -0.20639780163764954, -1.0293822288513184, -0.04543347656726837, 0.6794466376304626, 1.4238380193710327, 1.6401203870773315, -0.8871215581893921, -0.5876402854919434, -0.9789241552352905, -0.3594740331172943, 2.525789260864258, -0.2825031578540802, -0.11159234493970871, -1.5347371101379395, -0.17341558635234833, 0.66376131772995, -0.2541072964668274, 0.3714137673377991, 0.5576440095901489, -0.32063770294189453, -0.7561721801757812, 0.22048018872737885, 0.37998834252357483, 0.331348717212677, -0.8151035308837891, -0.5142143368721008, 2.3513832092285156, -1.6436563730239868, 0.5491020083427429, 0.2797156870365143, 0.9093599319458008, -1.455256700515747, -0.7091915011405945, 0.976537823677063, -1.429787278175354, -1.3881478309631348, -0.7429558038711548, 0.09417761862277985, 1.3973655700683594, -0.1184472143650055, 0.28417229652404785, 0.9897021055221558, 1.5030466318130493, -0.954302966594696, -0.6077892184257507, -0.8929057717323303, -0.49676042795181274, -0.699189305305481, -2.1183714866638184, 1.2221601009368896, -0.09603773802518845, 0.39776402711868286, -1.1302412748336792, -0.3337394893169403, 0.5681782960891724, -0.10153724253177643, -0.43628016114234924, 0.8783212900161743, -0.03566204383969307, -0.8970902562141418, -1.1756179332733154, -0.0855294018983841, -0.4273753762245178, 0.4056858420372009, -0.672263503074646, 0.6685261130332947, -0.12236586213111877, -1.743822693824768, -0.4136715233325958, -0.5742520689964294, -0.3254653811454773, -0.18344253301620483, -0.5004249215126038, 1.992004632949829, -0.41748982667922974, -1.5908430814743042, -0.14575912058353424, -1.0257350206375122, 0.876645565032959, 0.39520931243896484, 1.1189640760421753, -2.391867160797119, -0.48742806911468506, -0.6381484866142273, -0.489228755235672, -0.6372313499450684, -0.3798471987247467, -0.9659929871559143, 0.175714910030365, -0.9685811996459961, 1.6039185523986816, -0.8761404156684875, 2.2114758491516113, -0.508230984210968, -0.08304492384195328, -2.0723657608032227, -1.3066706657409668, -0.2319633662700653, 1.1706945896148682, -0.44251713156700134, -0.11278746277093887, 0.6175296902656555, 0.7041391134262085, -1.5784039497375488, 1.516908049583435, 0.607428789138794, 0.08083916455507278, 0.32229360938072205, 1.4780383110046387, 0.7424907088279724, -0.1258193850517273, -1.454978585243225, -0.38494181632995605, 0.10733138769865036, -0.3542235195636749, 1.9053131341934204, 1.1280168294906616, -0.2818835973739624, 0.3949689269065857, -0.5989546775817871, 2.170140504837036, 0.2770586907863617, 0.9112551808357239, 0.13178910315036774, 0.33440253138542175, 0.2969425618648529, -0.44023072719573975, 0.21168410778045654, -1.9719136953353882, 2.0607526302337646, 1.1279221773147583, -0.4414612650871277, 0.7197073698043823, 0.30941054224967957, 0.21882915496826172, 0.46325576305389404, -1.9500398635864258, 0.5586156249046326, 1.2217954397201538, 0.5163657665252686, -1.4573746919631958, 0.7002099752426147, 0.3781549632549286, 1.1861402988433838, 0.44942140579223633, 1.537408471107483, 0.5250835418701172, 1.0656996965408325, 0.5037598609924316, -0.1431434154510498, -2.166445016860962, 0.5647686719894409, 0.7054767608642578, 0.5145841836929321, 0.9684911966323853, 0.6123920679092407, 0.5588290095329285, 0.5065913796424866, -0.4009930491447449, 1.0750102996826172, -0.08911638706922531, -0.728803277015686, 1.4540079832077026, 0.959619402885437, 0.5906327962875366, 0.4308856129646301, -1.3004008531570435, -0.2749437093734741, -0.7434546947479248, 0.60714191198349, 0.21394367516040802, 0.7651234269142151, 0.2855387330055237, -3.213470458984375, 1.4224761724472046, -0.19096918404102325, -0.752457857131958, 1.2594056129455566, -0.4833157956600189, 1.2652742862701416, 0.5344833135604858, -0.663616955280304, 0.03265484794974327, 0.7275494933128357, 0.004195499233901501, -0.7327297925949097, 0.6861267685890198, 1.379681944847107, -1.2632062435150146, 1.0292474031448364, -2.206681489944458, 1.450312614440918, -0.7921091318130493, 0.2932587265968323, -0.15945656597614288, -0.8493070006370544, -0.5163634419441223, 0.25047820806503296, -0.8693645596504211, -0.6629505157470703, -0.14176858961582184, 0.3393471837043762, -0.13270244002342224, 0.3101852536201477, -1.4283854961395264, 0.38872116804122925, 0.43661096692085266, 2.1800990104675293, 0.6221327185630798, -0.5444777011871338, 0.45061755180358887, -0.552306592464447, 1.8029440641403198, -1.4247887134552002, 1.9208393096923828, 0.9357480406761169, -0.11234647780656815, 1.787752389907837, -0.4977963864803314, -0.36215856671333313, 0.26970309019088745, 0.4778514802455902, -1.457072138786316, 0.35076409578323364, 0.4012582302093506, 0.8607913255691528, -0.2938498556613922, -1.0822279453277588, -0.5916182994842529, 2.2152178287506104, -0.4959263503551483, 1.3352060317993164, -0.8887747526168823, -0.26115232706069946, -2.069040060043335, -0.08142777532339096, -1.637826919555664, 0.6661357879638672, -1.316062331199646, -0.3346441686153412, 0.2207193374633789, -1.267138123512268, 1.4946259260177612, -0.7604269981384277, -0.7631929516792297, -1.228344202041626, -1.5726596117019653, 0.582153856754303, 0.13031691312789917, 0.12015285342931747, -1.8449958562850952, -0.565677285194397, -0.36523574590682983, -0.2212272584438324, -0.5414730310440063, -1.503312110900879, 0.040574219077825546, 0.3912077844142914, 2.1263504028320312, 0.3062461018562317, -1.949289083480835, 0.685477077960968, -0.9263499975204468, -2.338346004486084, -1.2942992448806763, 0.515826940536499, -2.51292085647583, -0.7665100693702698, -0.9470933675765991, 0.1364820897579193, -1.1816672086715698, -0.3631596565246582, 1.2236653566360474, 0.1331416815519333, -0.3691978454589844, -1.6836636066436768, 1.9047966003417969, -0.1048356369137764, -1.4952298402786255, 1.4527034759521484, 0.5503354072570801, 1.2676331996917725, 0.856199324131012, -1.7349622249603271, -0.4559098780155182, -0.33898210525512695, -2.0989608764648438, -0.4508237838745117, 1.6523417234420776, -0.08674545586109161, 0.4410898983478546, 1.3799564838409424, 0.718808650970459, 0.6326279044151306, -0.008724871091544628, -0.8654254078865051, -0.02137148566544056, 0.5771684646606445, -0.6839906573295593, 0.44651949405670166, -0.784008800983429, 0.0876043289899826, -1.8200528621673584, 0.25680065155029297, 0.7639660835266113, 0.0252668634057045, -0.3362730145454407, 1.1023298501968384, 0.3193313181400299, 0.21536755561828613, -0.7615598440170288, -0.5071976780891418, 1.0421596765518188, 2.272885799407959, 1.0026766061782837, 0.6912485957145691, 1.463282585144043, -0.9342413544654846, 1.2021607160568237, -0.5908939838409424, 0.3238076865673065, 0.5016427636146545, 0.6190041899681091, -1.2861950397491455, 0.5856148600578308, -0.8341090679168701, -0.21181869506835938, -1.1762324571609497, 1.2111989259719849, -0.5908265113830566, 0.6771183013916016, 0.5057618021965027, 0.13665834069252014, 1.1910232305526733, 0.5614194273948669, -1.099599003791809, -0.9059268236160278, 0.5069054365158081, -0.18425975739955902, 0.8070744276046753, 1.0911438465118408, -0.0631360337138176, 0.10810555517673492, 1.218640923500061, -1.7862927913665771, 0.0950486809015274, -1.495273470878601, -0.3706856667995453, -0.037586335092782974, 1.2218266725540161, 0.6240055561065674, -0.005249443463981152, 0.3522840738296509, -0.6547719240188599, -1.8748265504837036, 0.9331533312797546, 0.6381168961524963, -0.3782612681388855, -0.32534855604171753, 0.41899222135543823, 0.0689140036702156, -0.6369776725769043, 1.2834054231643677, -0.09518308192491531, 0.5697507262229919, -0.909342348575592, 0.7750879526138306, -0.23732666671276093, 0.41572755575180054, -2.1249918937683105, -0.29792842268943787, 0.1548837125301361, -0.027313701808452606, 0.2638879120349884, 0.04862712323665619, -0.47060537338256836, -0.0665820762515068, 0.1782694309949875, 1.377407431602478, 0.9736971855163574, -0.7526931166648865, 0.42522525787353516, -1.348709225654602, -0.26707154512405396, -0.7258996367454529, -0.14609965682029724, -1.3720694780349731, 0.014259465038776398, -0.0010415693977847695, 1.7832759618759155, 0.08710967749357224, -0.43734368681907654, 1.519490122795105, 0.8482264280319214, 1.03258216381073, -0.8470402956008911, 0.8090531826019287, -0.14212019741535187, 1.0873173475265503, -1.6222532987594604, -1.1322543621063232, -0.5892102718353271, 0.382259339094162, -0.766608476638794, -0.5329226851463318, -1.8105597496032715, 1.1720988750457764, 0.4934712052345276, 1.3882259130477905, 1.1908926963806152, -1.0797520875930786, -0.2982444167137146, 0.03880602493882179, 0.641780436038971, -1.0926631689071655, -0.9301092028617859, 0.3906938433647156, -1.3510478734970093, -1.2232306003570557, -1.778059959411621, 0.797782301902771, -0.721733808517456, 0.4517238736152649, -0.12923173606395721, 1.895254373550415, -1.353027582168579, -0.704349935054779, 0.09617087990045547, 0.09519817680120468, -0.50494784116745, 0.7432997822761536, -0.35607731342315674, 2.1679255962371826, 0.13135497272014618, 0.07934550940990448, 0.8029310703277588, 0.3026832044124603, -0.6426993012428284, 0.5907403826713562, -0.1743726134300232, 1.9315437078475952, 0.6441658735275269, -0.7868809103965759, -0.3466504216194153, -0.39297953248023987, 0.2955591380596161, -0.7669996023178101, 1.339385747909546, -0.8088110089302063, 0.25497955083847046, 0.8177321553230286, -1.707362413406372, -0.9494526386260986, 0.5046756863594055, 0.9831164479255676, 0.3280288279056549, -1.272117018699646, 0.4299190640449524, -0.8442373275756836, -0.45481812953948975, -0.014171847142279148, -2.519041061401367, -0.8821973204612732, -0.9290485382080078, -0.07044760882854462, -0.1464528739452362, -0.9499304890632629, 0.9806616306304932, 0.3930486738681793, -0.9142341017723083, 0.7372256517410278, 1.7440814971923828, -1.8564329147338867, -0.13261155784130096, 1.9766367673873901, -0.6514313817024231, -0.36811375617980957, 1.0190223455429077, 0.1732146143913269, -1.0740607976913452, -0.32652416825294495, -1.103489637374878, -0.844302773475647, -0.27123233675956726, 0.3667253255844116, 0.8590246438980103, -1.275746464729309, -0.9986649751663208, 0.8063113689422607, -0.11742297559976578, 1.0648044347763062, -1.4430971145629883, 0.1532323658466339, -0.3644959330558777, 1.1541792154312134, -0.19155877828598022, 0.5263151526451111, 0.36550435423851013, -1.9191062450408936, -0.9441219568252563, -0.45173513889312744, 0.8274760246276855, -0.61882084608078, 0.24185986816883087, -0.44977203011512756, 0.7585897445678711, -0.0217721126973629, 0.607994794845581, 1.0794563293457031, -1.0805433988571167, 0.6657481789588928, 0.16062122583389282, -0.1272576004266739, 0.024034587666392326, 0.9196688532829285, 0.10902252793312073, -0.6150073409080505, 0.04178949072957039, -1.507215976715088, -1.6376943588256836, 0.9191361665725708, -0.008317936211824417, 1.060939073562622, -0.4453841745853424, -0.7991330027580261, -1.3564105033874512, -0.08960651606321335, -0.4224182963371277, 0.011523578315973282, -1.0311336517333984, 1.1796084642410278, -1.6276310682296753, -0.04585753008723259, -0.8458133339881897, 1.0081512928009033, 1.1725093126296997, 0.4925253093242645, 1.3330425024032593, -0.8883762955665588, -0.6252750754356384, 1.5862261056900024, 0.03444643318653107, 0.874566376209259, -1.4085628986358643, -1.4097034931182861, 0.9507295489311218, 0.5843412280082703, 0.27137982845306396, -0.41294220089912415, 0.429080605506897, -0.16516655683517456, 0.45036524534225464, -0.8468284606933594, 0.8405659794807434, 1.1417804956436157, -0.07141394168138504, -0.4771777391433716, 0.040263086557388306, 0.940847635269165, -0.7741263508796692, 1.3723623752593994, 0.4885052442550659, -0.6574026346206665, 0.3015406131744385, 2.4041662216186523, -1.3043419122695923, 1.054210901260376, 1.3719037771224976, 0.0400267131626606, 0.09607575088739395, -2.104750156402588, -1.5726990699768066, 0.7495999932289124, -0.2073426991701126, -0.674217700958252, 0.18033143877983093, 1.857401728630066, 0.3469526767730713, -0.36931151151657104, -0.24948297441005707, 0.5777361989021301, 0.6993899941444397, -0.4130629003047943, -0.9634233713150024, -0.07064829021692276, 1.0266400575637817, 1.2491272687911987, -0.4342190623283386, -0.3346068263053894, 1.9885503053665161, 0.8255637884140015, 1.143857479095459, -0.997052013874054, 0.06119858846068382, -1.7040008306503296, -0.19901888072490692, -0.3532842993736267, 0.23950548470020294, 0.7520865797996521, 1.6334195137023926, -2.3442559242248535, -1.4367427825927734, -1.7339681386947632, -1.5591225624084473, 1.1233632564544678, 1.6409741640090942, 0.9033753871917725, -1.2769858837127686, 0.08108463883399963, 1.0999757051467896, -2.2775800228118896, -1.7926762104034424, -1.047433853149414, -0.017357442528009415, -1.2735686302185059, -1.4314539432525635, 1.2431092262268066, -0.653529167175293, 0.10416994988918304, -0.15009954571723938, 0.8587087988853455, 0.9121181964874268, 1.298184871673584, -0.9454838037490845, -0.49785569310188293, 0.13176685571670532, 1.8626203536987305, 0.2148941457271576, 0.7263118624687195, 1.1337835788726807, 0.9047887325286865, -0.3809141516685486, 0.8392322659492493, 1.3961114883422852, -0.9447148442268372, 0.8053995370864868, 0.9554463624954224, -0.5488674640655518, 0.49917662143707275, -0.10501336306333542, -2.2328944206237793, 0.628771185874939, 0.4089723527431488, 0.9727323055267334, -0.7223614454269409, -0.8095497488975525, 0.275812029838562, -0.7481323480606079, -2.640456438064575, 0.8347896933555603, -1.7653299570083618, 1.0491738319396973, 2.439225196838379, 0.8490726351737976, 0.535047709941864, -0.24141213297843933, -0.45105603337287903, -1.1170135736465454, -0.8921529054641724, -0.9792988896369934, -0.1773620843887329, -0.1753775030374527, 0.12563163042068481, 0.14063306152820587, 1.3114278316497803, 1.0955606698989868, 0.48158973455429077, 0.32970425486564636, -2.7894458770751953, 0.0975252240896225, -0.9992368221282959, -0.6608004570007324, -1.0835208892822266, -0.5004698634147644, 0.5778641104698181, -0.03971482440829277, -0.8887538909912109, -0.6218320727348328, -0.47261592745780945, -1.653328537940979, 0.002669072011485696, 1.0559899806976318, -1.292183518409729, 0.6152396202087402, -0.013490107841789722, 1.7339386940002441, -1.0821385383605957, -0.5983039736747742, 0.6162691116333008, -0.05783836543560028, 1.1049509048461914, 1.8637683391571045, -0.9363305568695068, 1.362524390220642, 0.1428319215774536, 0.2585359215736389, -0.23748962581157684, 0.0852397084236145, 0.6453532576560974, -0.7038631439208984, -0.19973845779895782, 0.47160476446151733, -0.32203730940818787, -0.4813069701194763, 0.7587088942527771, 1.9347825050354004, 0.3610146939754486, 0.8002716302871704, 0.16241413354873657, 0.457634299993515, 2.061997652053833, -0.328335165977478, 0.23561115562915802, -0.23475609719753265, 1.7410627603530884, 1.1981620788574219, -1.1574294567108154, -0.5526400804519653, 0.06230951100587845, -0.2934879958629608, 0.528080940246582, -0.3222868740558624, -2.4762978553771973, -0.04658344388008118, -1.2037588357925415, 0.24613414704799652, 1.0725706815719604, 1.1650229692459106, -1.4983489513397217, 0.9893772006034851, -0.5105969309806824, 0.6676452159881592, -0.03746127709746361, 1.3873857259750366, -0.8261709809303284, 1.2661736011505127, 0.5007143616676331, 1.4263861179351807, -0.48584967851638794, -2.1228935718536377, -0.8893417119979858, -1.406891942024231, 0.5210933089256287, 1.3061755895614624, 0.5413411259651184, -1.5503344535827637, 1.8300267457962036, -1.2923932075500488, 1.7280939817428589, 0.2773691415786743, 0.22721871733665466, -0.5711365342140198, -0.16704009473323822, 0.7080813050270081, -0.22412380576133728, -0.18239907920360565, -0.16862531006336212, 1.8317246437072754, 1.9580765962600708, -1.402592420578003, -0.5124240517616272, 0.4350067377090454, 1.2061349153518677, 2.443086862564087, -0.6338639259338379, -0.0872257724404335, -0.2000035047531128, 1.2829058170318604, -2.200340509414673, -1.4845597743988037, 0.44338351488113403, 1.5377126932144165, 0.16372910141944885, 0.28374624252319336, 0.5976756811141968, 0.438953161239624, -0.265036940574646, -0.3531705141067505, 0.40454530715942383, 0.8645531535148621, 0.5505862236022949, -0.34843429923057556, 0.48176369071006775, 0.7427790760993958, -0.3895083963871002, 1.0286810398101807, 0.4228847622871399, -0.04713129997253418, 0.5681137442588806, -0.6320536136627197, -1.3313140869140625, 0.8778213262557983, -2.026825189590454, -1.0640766620635986, -0.39326223731040955, -0.7957789897918701, 0.2607908248901367, -0.34519627690315247, 1.0688637495040894, -1.349723219871521, 0.4633442461490631, -0.9233297109603882, -0.6723393797874451, -0.1034366562962532, -1.4700714349746704, 0.7587817907333374, 0.2722150683403015, 1.147265911102295, 1.2579630613327026, 0.2471880316734314, 0.7292146682739258, -1.473167061805725, -1.5192666053771973, -0.7187477350234985, 1.552032709121704, -2.2660794258117676, 1.406737208366394, -0.8433098196983337, -0.024540182203054428, 0.8108293414115906, 1.242283582687378, -0.5319521427154541, -0.6866702437400818, -0.8610590100288391, 2.905127763748169, 0.7594136595726013, -1.7801551818847656, 0.3254300355911255, 1.3696386814117432, 0.6174347400665283, -1.24570631980896, -1.9738062620162964, 0.0095462491735816, 0.7861995697021484, -0.22221125662326813, 0.22160084545612335, 0.22428281605243683, 0.1421194076538086, 0.813974916934967, -2.17685866355896, 0.4339492619037628, 0.7725878357887268, -0.6802474856376648, -0.46814969182014465, 1.918095350265503, -0.45654726028442383, 1.213195562362671, -0.4046711325645447, 0.7809337973594666, 2.1270031929016113, 0.717559814453125, -0.21576330065727234, -0.41587936878204346, 1.4471054077148438, 0.5308093428611755, 0.7483706474304199, 2.4237358570098877, 1.835037350654602, 0.23256200551986694, 0.7338374853134155, 0.13423258066177368, 0.6352466940879822, 1.272946834564209, -0.4077467620372772, 0.8499961495399475, 0.5860419869422913, 0.09470860660076141, 0.6788793206214905, 1.4927502870559692, 1.2607440948486328, -0.8551554679870605, -0.01965561881661415, -0.08835635334253311, -0.17221784591674805, -0.43109309673309326, -1.7821489572525024, 0.1952362060546875, -0.02451816014945507, -0.5297319889068604, 0.3046616315841675, 0.43998879194259644, -0.1228877454996109, 2.1554901599884033, 0.08284693211317062, 0.40226733684539795, -0.28252267837524414, -0.10532652586698532, -1.1687651872634888, -0.06189039722084999, -2.0618581771850586, 2.487706422805786, 0.5985121130943298, -0.49635085463523865, -0.7222882509231567, -0.405666321516037, -0.35805073380470276, 0.20653876662254333, -0.4404613971710205, -1.0605077743530273, -0.5573891401290894, 1.2875498533248901, 1.2184869050979614, 0.0842384621500969, -1.1861791610717773, -0.5217275023460388, -0.6195226907730103, -0.5790805220603943, 0.165536031126976, 0.23537203669548035, 0.13335870206356049, -0.04480092227458954, 0.5287370681762695, 0.7831632494926453, -0.6319740414619446, -0.888802707195282, 0.5781866908073425, 1.161036729812622, 0.4415484368801117, -0.015353907831013203, 0.9341063499450684, -0.34127986431121826, 1.3465067148208618, 0.3364778161048889, -0.387100487947464, -1.318901777267456, -2.561598539352417, 0.12465625256299973, -0.22847019135951996, 0.9405401945114136, 0.41605404019355774, 0.8817896246910095, 0.5459644198417664, 1.5928641557693481, -0.7341368198394775, -0.7873890995979309, 0.2505817413330078, -0.4158230125904083, -1.444782018661499, -0.08224044740200043, -1.191999912261963, 0.029057659208774567, 1.5775254964828491, -1.2418025732040405, 0.9595915675163269, 0.573957622051239, -0.8368076086044312, -0.4944051206111908, -1.3397270441055298, 0.34827473759651184, 0.6098055243492126, -1.0668376684188843, 0.5464615821838379, 0.21575066447257996, 0.9847743511199951, 0.2194308638572693, -0.5824136137962341, -0.3673972487449646, 0.3978841304779053, -0.33048713207244873, 0.09918085485696793, -0.7852253913879395, 0.8379542827606201, 0.21015572547912598, 1.2541706562042236, -0.7164599895477295, -0.9470654726028442, 1.1573939323425293, -0.10113172978162766, -0.29355666041374207, 0.4426422715187073, -0.127893328666687, 1.1488348245620728, 1.9793882369995117, 0.32468655705451965, -0.8837957978248596, 0.18966034054756165, 0.3851972818374634, -0.828859806060791, 0.3969168961048126, 2.0658044815063477, 1.1662017107009888, 0.012870593927800655, -0.08090510964393616, -0.38761189579963684, -1.387949824333191, -1.1932880878448486, 0.6503753066062927, -0.2820696532726288, 1.4909147024154663, 0.4534584581851959, -0.436083048582077, -0.1526915282011032, -0.39748722314834595, -1.5407932996749878, 0.4665827751159668, -0.0618259534239769, 1.9803208112716675, -2.17873477935791, -0.7111803889274597, 0.9680312871932983, -0.24791128933429718, -0.005330166779458523, 1.066956877708435, -0.39991775155067444, 0.9612692594528198, 0.6943974494934082, 1.184765100479126, -1.511318325996399, 1.1287708282470703, -0.18769888579845428, 0.02993873693048954, -1.0554648637771606, 0.3005579710006714, 0.5849748253822327, 1.3871570825576782, -1.225555419921875, 1.2606667280197144, -0.11512947082519531, 0.1636340469121933, -1.8744837045669556, 1.244600534439087, -0.8988494873046875, -0.1183985248208046, 0.24895061552524567, -0.17049221694469452, -1.3395678997039795, 0.9197107553482056, -1.072165846824646, -0.16350948810577393, -2.244192600250244, 0.26106104254722595, 1.586755633354187, 0.6822983026504517, -0.34053316712379456, -1.0352131128311157, -0.23467391729354858, 0.15263380110263824, -1.2096366882324219, -1.880989670753479, -1.0820164680480957, 1.9477542638778687, 1.1776808500289917, 0.8565329909324646, 0.4351276159286499, -0.6021087169647217, -0.4492504894733429, 1.0637850761413574, -0.795056164264679, 0.31654831767082214, 2.6250321865081787, 0.06554833054542542, 1.2067538499832153, 0.6118236184120178, 1.583125352859497, -0.6265486478805542, 0.3007879853248596, -2.3248696327209473, 0.014657443389296532, -0.021997570991516113, 0.14464831352233887, 0.022637590765953064, 0.4848368167877197, 0.4701478183269501, 3.2409188747406006, -1.2265918254852295, 0.5704697966575623, -1.2344743013381958, -0.4687110483646393, 0.5997796058654785, 0.04402056708931923, 0.3068350553512573, 0.47271692752838135, 0.31990760564804077, -1.0543560981750488, -0.10651063174009323, 2.0215871334075928, 0.6164292693138123, -1.4218764305114746, 1.481833577156067, 1.26203453540802, 0.2104726880788803, 0.21394291520118713, 0.5344962477684021, 0.8273873329162598, 1.0666937828063965, 1.049147367477417, -1.328817367553711, -0.9131861329078674, 1.8181695938110352, 0.9924131631851196, -0.7569776773452759, 1.9591021537780762, 1.3834342956542969, 0.10905437916517258, 1.1353703737258911, 1.166355013847351, -0.46298158168792725, 1.0750564336776733, -0.2144434005022049, 0.6189632415771484, 0.9559336304664612, 1.178324580192566, -0.45242995023727417, 0.4628710150718689, 0.9461251497268677, -1.3887739181518555, 1.1424492597579956, 1.8495086431503296, -1.030929684638977, 1.4136788845062256, 1.403048038482666, -1.0326894521713257, -0.011136554181575775, -1.9687426090240479, -0.552513837814331, -1.4994046688079834, 1.9893481731414795, -0.8907641768455505, -0.811352550983429, 0.44871512055397034, 1.3136235475540161, -1.9707683324813843, -0.35748618841171265, 0.9584842920303345, 0.22599345445632935, 0.3266874849796295, 0.612664520740509, 0.039056070148944855, 1.8977851867675781, 0.6521297693252563, -0.4099942743778229, -2.2322216033935547, -0.22531113028526306, 0.3441391885280609, -1.3453011512756348, -0.526643693447113, -1.418191909790039, 1.4161299467086792, -1.3513284921646118, 1.4525723457336426, -0.36305540800094604, 0.9831089973449707, 0.334364652633667, -0.6389751434326172, 0.5247248411178589, 0.018217332661151886, -0.5384194254875183, -0.7241488099098206, -1.0538254976272583, -1.575827956199646, -0.8902695775032043, -0.42972904443740845, -0.46072104573249817, -1.4686845541000366, 0.07472606748342514, 0.028519796207547188, -1.4627020359039307, -0.5315730571746826, 0.2869783341884613, -0.3029235899448395, 0.21696263551712036, -0.4643110930919647, -0.5377657413482666, -1.636710524559021, -0.020656611770391464, -0.0025654800701886415, -0.36195361614227295, 1.8073811531066895, 1.867785096168518, -1.5010337829589844, -0.03963306546211243, -0.9656595587730408, -1.444714069366455, 1.9989371299743652, -1.1241506338119507, 1.32423734664917, 0.6508930325508118, 0.627582311630249, -0.5532079935073853, -0.6318876147270203, 0.6063042283058167, 0.4000799357891083, -0.6727504134178162, 0.1780557632446289, -0.026577504351735115, -0.3471600115299225, -1.3657760620117188, -0.2541077733039856, -0.17154762148857117, -0.4459024667739868, 0.47975555062294006, -0.486404687166214, -0.08788151293992996, 0.9355140924453735, -0.8508939743041992, -0.41953709721565247, -1.0623449087142944, 0.22254998981952667, 0.7945089936256409, -1.1558409929275513, 0.27114006876945496, 0.15345384180545807, -1.046402931213379, -0.43614768981933594, -2.745739698410034, 2.5374817848205566, 0.4638332426548004, 1.3526548147201538, -0.2221517711877823, -0.1561540961265564, 1.2898883819580078, -0.845718502998352, 0.7079079151153564, -0.19623322784900665, 0.23973745107650757, 0.024631941691040993, -0.015103868208825588, -0.28251224756240845, 2.6831247806549072, 1.1427918672561646, 1.9690228700637817, 0.45680639147758484, 0.05400216951966286, -1.791674017906189, -0.09922011196613312, -0.7861368060112, -0.049910735338926315, -0.3746342957019806, 2.389399290084839, -1.406912088394165, 0.18164213001728058, -1.118872046470642, -0.9950167536735535, -0.7751940488815308, 0.39869722723960876, 0.34489312767982483, -1.4785099029541016, 1.4254251718521118, -0.42192429304122925, -0.29118844866752625, -0.7807808518409729, -0.11067907512187958, 0.5381201505661011, 1.838455319404602, -1.0173710584640503, 2.6271955966949463, -0.07460620254278183, -1.5977718830108643, 0.05410675331950188, 0.3402785360813141, -2.022907018661499, -0.2503100335597992, 1.205966830253601, 0.43195250630378723, 0.6266142129898071, 0.5768083930015564, -0.2851514518260956, -0.020556749776005745, -1.1684510707855225, 0.9426329731941223, -2.6717448234558105, 0.017851078882813454, -0.981257975101471, -1.1067794561386108, -0.06785210222005844, 0.609627366065979, -1.1941882371902466, 1.4862854480743408, 0.15303586423397064, -0.1791813224554062, -0.9165905117988586, -0.8381608128547668, 1.6287357807159424, -1.0994584560394287, -0.3833663761615753, -0.8521141409873962, 1.0187886953353882, 0.3690566420555115, -0.4101767838001251, -0.3664219081401825, 1.664983868598938, 0.12338626384735107, -1.2536541223526, -0.7822005152702332, -0.4074968993663788, -1.8064930438995361, 0.9102291464805603, 0.6510496139526367, -0.7482232451438904, -0.43156927824020386, -0.26752686500549316, -0.41442349553108215, 1.7038625478744507, -0.34867429733276367, 0.17102378606796265, 0.8410898447036743, 0.646052896976471, 1.2459979057312012, -0.6229852437973022, 0.35702094435691833, -0.9052676558494568, 1.1878113746643066, -0.7577961087226868, 1.4326510429382324, -0.8386656045913696, -0.5552427172660828, -0.5724842548370361, -1.1999012231826782, 2.2140164375305176, -0.29502853751182556, -0.8156412839889526, -1.213705062866211, -0.3784426152706146, 1.6613314151763916, 1.6362276077270508, 0.30661603808403015, 1.0655990839004517, -0.11327510327100754, 0.4678882956504822, 0.015882546082139015, 0.7070944905281067, 1.7200241088867188, -1.0607424974441528, 0.8865143656730652, 0.42960894107818604, -0.2010609656572342, -0.2787277400493622, 1.5052642822265625, 0.8880102634429932, 1.226293921470642, 0.47464948892593384, -1.2931355237960815, 0.5168564319610596, -0.8007251024246216, 0.3426947295665741, -0.18091121315956116, 1.9284688234329224, 0.3967062532901764, -1.2238987684249878, 0.009993131272494793, -0.026141351088881493, 1.4551877975463867, 0.03234457969665527, -0.5801247954368591, 0.824286699295044, 1.6121654510498047, 0.0333547405898571, 0.837902843952179, -1.0498443841934204, 0.07378851622343063, -1.340678095817566, -0.4545327425003052, 0.8859469890594482, 0.27718114852905273, 0.43179869651794434, -0.15820792317390442, -1.3145828247070312, -0.779442548751831, 0.5294411182403564, 0.1358349472284317, 1.124118447303772, -1.062394618988037, -0.8555832505226135, -0.6304110884666443, 0.43191397190093994, 1.119384765625, 1.10215425491333, 0.03193828463554382, -0.3103272318840027, 0.8439398407936096, 0.37607133388519287, 1.5547493696212769, -0.8137267827987671, 0.17480939626693726, 1.475942850112915, 0.33162468671798706, -0.17999763786792755, -0.8644890189170837, 0.011548381298780441, 0.23325097560882568, -0.6834632754325867, 0.7178871035575867, 0.2929929792881012, -1.6316978931427002, -0.6720182299613953, -1.576776385307312, -0.8404017686843872, -0.1270565688610077, -1.0201168060302734, 0.3393888473510742, -2.940349578857422, -0.8613499999046326, 1.2076128721237183, -0.354874849319458, -0.6835736036300659, -0.34358587861061096, -0.17052650451660156, -0.3593989610671997, -0.5592437386512756, -1.8023453950881958, -1.108292818069458, -1.339997410774231, -0.5779709815979004, 0.6716815829277039, 0.6078506708145142, 0.192783921957016, -0.760431706905365, -0.17313431203365326, -0.4488323926925659, 0.18382544815540314, 0.38251951336860657, 0.7927488684654236, -0.17102496325969696, 0.29031306505203247, 1.613599181175232, 0.5917073488235474, -2.744523048400879, 2.6055545806884766, 1.5887572765350342, -0.23529109358787537, -1.9786611795425415, -1.0846244096755981, 0.5222560167312622, -0.03709331899881363, 0.7332192659378052, 1.0466392040252686, 0.20614862442016602, -1.0781636238098145, 0.026760021224617958, -0.5329557061195374, 0.8483661413192749, -0.5800006985664368, 0.7511496543884277, 1.8545247316360474, -0.02102835476398468, 1.8422118425369263, 1.1135023832321167, -0.9128980040550232, -0.8937453031539917, 0.38846468925476074, -0.3770153522491455, 0.7483986616134644, -1.5864022970199585, 1.0638619661331177, -0.4597260057926178, -0.1126888245344162, 0.4665948450565338, 0.35119539499282837, -1.3099240064620972, 1.5844237804412842, 0.9611900448799133, 0.9190323948860168, -0.4243564307689667, 0.9748731851577759, 0.7567955851554871, 0.828719973564148, 0.9569839239120483, -0.3143792748451233, 0.7474172115325928, -1.184374213218689, -2.036565065383911, -2.078352451324463, -0.1188628152012825, -0.7763634324073792, -0.17092907428741455, 0.09580922871828079, -0.4816287159919739, 0.9251320958137512, 0.3003864884376526, 0.3346061706542969, 1.790971040725708, 0.42490217089653015, -1.8370692729949951, -1.4272425174713135, -0.7656466364860535, -0.8210130333900452, -0.7724904417991638, 0.829735279083252, -0.5050426125526428, -0.08554556220769882, -0.4865712523460388, 1.8062149286270142, 0.3950605094432831, -0.548823356628418, -0.23436179757118225, -1.3398998975753784, -1.4205390214920044, -0.4996534287929535, 0.5188291072845459, 0.26019540429115295, 1.4421390295028687, 0.6951736211776733, 0.47094109654426575, -1.1344718933105469, -1.3383643627166748, 0.36688491702079773, 0.5536511540412903, -0.15138912200927734, -0.8446213006973267, 1.6021554470062256, 0.1540692150592804, 0.32289618253707886, 1.9212255477905273, 0.7753527164459229, 0.4889228343963623, 0.8324379324913025, -1.0195969343185425, 0.8160125613212585, 0.2767462134361267, -0.4657990634441376, -0.9200230836868286, -0.6417032480239868, -0.78059321641922, -0.8455002903938293, -2.2540218830108643, 1.527661681175232, -0.31695467233657837, -0.5206583738327026, -0.0661618784070015, 0.8154194355010986, 0.19001072645187378, 1.3099485635757446, -0.7324672937393188, -0.07263496518135071, -0.7515689134597778, 1.7801915407180786, -0.5268753170967102, 1.5892810821533203, 1.1328825950622559, 0.44065117835998535, 0.7536563277244568, 1.537387728691101, 0.9438184499740601, -1.922340989112854, 0.2833237051963806, -1.9147192239761353, -0.2884284257888794, -0.01221719291061163, 0.22419767081737518, 0.39413055777549744, -0.6414023637771606, 0.05294059216976166, 0.8453688025474548, 1.9796757698059082, -2.1119046211242676, 0.1892831027507782, -0.37967193126678467, -0.4595217704772949, 2.2982044219970703, -1.8284307718276978, 2.1812634468078613, 0.09102759510278702, 0.9998756647109985, 0.3773758113384247, 1.575871229171753, 0.3676314055919647, 0.3339449465274811, 0.10992339253425598, -0.9689491987228394, -0.28823211789131165, -0.07740732282400131, -1.028975009918213, 1.201372504234314, 2.190711259841919, 0.5998905897140503, 1.2707524299621582, 0.4966755211353302, 1.4800775051116943, -1.1184992790222168, 0.8091420531272888, -0.6260871887207031, -0.4369068443775177, 0.39854058623313904, -0.8781027793884277, -0.5608105063438416, -1.2352083921432495, 0.3242195248603821, 0.3584345579147339, 0.028631728142499924, 0.18800850212574005, 0.293467253446579, 1.5733298063278198, -0.12819431722164154, 0.06772574037313461, 1.3784211874008179, 1.3250274658203125, -2.208857536315918, 0.20223577320575714, -0.5076085329055786, -0.24228046834468842, -1.5759143829345703, 0.5988273024559021, -1.8599106073379517, -0.7222655415534973, -0.7460405230522156, 0.05697481706738472, -1.1475303173065186, 0.23206108808517456, -2.0672669410705566, 1.2642582654953003, -0.6684097647666931, 1.527703046798706, -0.7290457487106323, 0.795118510723114, -0.5311981439590454, -0.6810321807861328, 0.26034611463546753, 0.2410648912191391, -0.21496866643428802, 0.739345908164978, 0.3731389343738556, -0.5771975517272949, -1.4941349029541016, -0.3317999243736267, -0.5792302489280701, 0.23699748516082764, 0.3390205204486847, -0.6644030809402466, -0.6291189193725586, -0.5225721001625061, 0.3987048268318176, -0.5416017174720764, 0.5631429553031921, -0.1591014713048935, -0.7340505719184875, -0.7803453207015991, -1.22798752784729, 1.5908254384994507, 0.4094710350036621, -0.38517966866493225, 1.0362907648086548, 1.6490243673324585, -1.0791947841644287, 0.7425415515899658, -0.6864520311355591, -1.6971136331558228, -2.412442445755005, 0.16491656005382538, -1.034313440322876, -0.731137752532959, -0.6714189052581787, 0.30024778842926025, -0.535843551158905, -0.8368844985961914, 0.9824042320251465, -0.4207755923271179, -0.7942026853561401, -0.14231416583061218, -0.6094852089881897, -0.7068932056427002, -1.0080050230026245, -0.03225456923246384, -1.5466017723083496, -0.6579403281211853, 1.0543774366378784, -0.16408808529376984, 1.1546634435653687, -1.4261341094970703, 0.3249885141849518, 0.8426974415779114, 1.730231523513794, -1.392463207244873, -0.9178480505943298, -1.2149479389190674, -0.8391804099082947, 0.4456554055213928, -0.17923158407211304, -0.9975008368492126, 1.9017738103866577, 1.304049015045166, 0.3163960874080658, -1.3080633878707886, 0.4233863949775696, 1.011682152748108, -0.47376909852027893, -1.462187647819519, -1.6164096593856812, -0.7243456244468689, -1.5986154079437256, 0.6810967326164246, -0.45249465107917786, -0.24090991914272308, 0.005566189996898174, 0.02196415513753891, -0.026764223352074623, 1.7868388891220093, -0.6383883357048035, 0.14787547290325165, -0.43570753931999207, -0.8224902749061584, -1.5054447650909424, -1.0268282890319824, 0.0008871345198713243, 0.2605888545513153, -1.2955808639526367, -0.9897173047065735, -0.24637185037136078, 1.0323196649551392, -0.5195854902267456, 0.15241196751594543, 0.7195191383361816, -0.4075876474380493, -1.4915738105773926, 0.14047811925411224, 0.6802131533622742, 0.4985908269882202, -1.081011176109314, 0.3780516982078552, 0.05179383233189583, -0.26303091645240784, 0.6951932907104492, -0.14709612727165222, 1.335882306098938, 0.6558697819709778], [-0.9121826887130737, -0.17244282364845276, -0.9608579277992249, 0.048558253794908524, -0.4519525468349457, 0.8969802260398865, -1.0902973413467407, 0.7950940132141113, 1.0509254932403564, -0.12384374439716339, -1.3830857276916504, 0.10727745294570923, -0.4752323031425476, 0.246119886636734, -0.08602441847324371, -0.39672139286994934, -0.026591628789901733, -0.4025081396102905, 1.0334227085113525, 0.05255494266748428, -1.7678581476211548, -0.684449315071106, 0.5095065236091614, 1.1711993217468262, 1.3121824264526367, -1.8299733400344849, -0.5213330984115601, 0.8560393452644348, -0.9155547022819519, -0.6385754942893982, 0.24786405265331268, -0.19848407804965973, 0.8878269791603088, -0.4047934412956238, 1.3804618120193481, 0.14013180136680603, -1.4264578819274902, -0.09585414826869965, -1.5779441595077515, -0.13617905974388123, -0.11852048337459564, 0.5521327257156372, 0.371822714805603, -0.24306419491767883, -0.3249388337135315, 2.1319148540496826, 1.1496890783309937, -1.3920855522155762, -0.006504050921648741, -0.5039761662483215, 1.4932359457015991, -0.1783789098262787, 0.32943058013916016, 0.35601380467414856, 1.9114744663238525, 1.5338547229766846, 0.4265632927417755, -2.103907346725464, 0.010547326877713203, 1.1134064197540283, 0.4622163772583008, 0.19044654071331024, 0.614010214805603, 1.6480354070663452, -0.7484886050224304, 1.3217848539352417, -1.9022035598754883, -0.5831174850463867, -0.5561760067939758, 0.14422065019607544, -0.8585559129714966, 0.2764526903629303, 0.08762791007757187, -0.7052987813949585, 0.7890673875808716, 0.8632839918136597, 0.18013271689414978, 0.9293789267539978, -0.8433728218078613, 0.5550236105918884, 0.8825941681861877, -0.34557873010635376, -0.5526426434516907, 0.7148119807243347, 1.8698012828826904, 0.13445788621902466, 0.9059278964996338, 1.3362146615982056, 0.9059162735939026, -0.8121560215950012, 1.4333393573760986, -0.24967612326145172, -0.1563837230205536, 0.7952978014945984, -0.012874715030193329, 1.2745082378387451, -0.036295391619205475, -1.2768925428390503, -0.5848305821418762, 0.9788326621055603, -0.5905259847640991, 0.6802241802215576, -1.1018105745315552, 0.5328807830810547, 0.16704939305782318, -0.4026320278644562, -0.8067885041236877, 0.6152202486991882, -0.8491357564926147, 1.0854747295379639, -0.4472883939743042, 1.528123140335083, -0.49936920404434204, 0.3115019202232361, 2.385695457458496, -0.02583830989897251, 0.8080080151557922, 2.060483694076538, -0.4054988920688629, -0.31693220138549805, -1.2885583639144897, -0.4277384281158447, -1.9138782024383545, 0.09852532297372818, 0.0003351099730934948, 0.06199704483151436, -1.4067898988723755, -0.40131932497024536, 0.634429931640625, 0.23687131702899933, 0.5658495426177979, -0.00990372896194458, -1.7940341234207153, -0.44187331199645996, -1.5028572082519531, 0.1583610326051712, -0.7260819673538208, 0.17349785566329956, 0.177369624376297, -0.34840089082717896, 1.284523367881775, 0.9829909801483154, -1.4097464084625244, 0.31848397850990295, -0.7081756591796875, -0.016890252009034157, 0.6001917123794556, 0.201040118932724, -0.30678582191467285, 1.3207271099090576, -0.586687445640564, -1.1052203178405762, 0.8535748720169067, 1.3372035026550293, 1.2267986536026, 1.6693974733352661, -1.501552700996399, -0.737399697303772, -1.9107784032821655, -2.0006706714630127, -2.0054726600646973, -1.1518051624298096, -0.2972821891307831, 1.2602688074111938, -0.910682201385498, 0.03931237757205963, -1.2148758172988892, -0.029833192005753517, 0.15933440625667572, -0.3507225811481476, -0.21940942108631134, -0.16972661018371582, -1.2575902938842773, -0.897773027420044, -0.7477442622184753, 0.6400147080421448, 0.5599169731140137, -1.2945691347122192, 2.3869965076446533, -0.15810281038284302, 0.11324705183506012, -0.6339914202690125, 0.8863418698310852, -1.5209600925445557, -1.7650045156478882, -0.8281183242797852, -2.4043707847595215, 0.7668505311012268, 1.7639517784118652, 0.36828523874282837, 1.0727627277374268, 1.0339707136154175, -1.945034146308899, 0.23319393396377563, -0.36718854308128357, 0.6594811081886292, 0.7695497870445251, 0.34858837723731995, -0.028665341436862946, 0.6640381813049316, -0.14609874784946442, 0.5794416069984436, -0.2448759526014328, 0.9541418552398682, 2.0089375972747803, 0.4665742516517639, 0.015639949589967728, 1.014845848083496, 1.4909415245056152, -3.445484161376953, -0.4048565626144409, -0.07953666895627975, -0.6511891484260559, 0.46911972761154175, -2.005620002746582, 0.19326719641685486, -0.600041389465332, -1.1026813983917236, 1.4247289896011353, -0.9554463624954224, -0.010497387498617172, -0.7085493206977844, 0.387527734041214, -0.08302921056747437, 0.33245882391929626, -0.14522884786128998, 1.566628336906433, -0.22953654825687408, -0.6960054039955139, -2.279587984085083, -0.6149078011512756, -0.8086186051368713, -0.03616539016366005, 0.23248717188835144, -0.15420164167881012, 1.896199107170105, 0.1189970076084137, 0.963382363319397, 1.5210769176483154, -1.1263381242752075, -0.9343909025192261, -0.22257216274738312, -1.2729426622390747, -0.0026533992495387793, 0.15550729632377625, -0.3596242666244507, 2.125541925430298, 0.7517556548118591, -0.031044287607073784, 0.39166030287742615, 0.7541354298591614, -1.5368680953979492, -0.3355054259300232, 1.0388814210891724, 0.4297546446323395, 1.4892934560775757, -1.0301684141159058, -1.0078359842300415, 0.3007408678531647, 0.9832446575164795, -0.25628167390823364, -0.021797209978103638, -1.4473294019699097, -0.10901310294866562, 1.2304283380508423, -0.6372281312942505, 0.6832461953163147, -0.02969183214008808, -0.2520069479942322, -0.5566167235374451, 1.154710292816162, 0.003836217802017927, -1.4439265727996826, -0.0022481416817754507, 1.4603362083435059, 0.1666257083415985, 0.8326292037963867, 0.02529873140156269, -0.750062108039856, -1.482865333557129, -1.3381181955337524, 0.8399059176445007, -0.5495408177375793, -1.214595913887024, -0.7263104915618896, 0.25718599557876587, -0.9645059108734131, -0.05119486525654793, 1.8824609518051147, -0.13200922310352325, -0.5041489005088806, 0.99658203125, -1.2562483549118042, -1.9779058694839478, -0.33210739493370056, -1.4981920719146729, 0.35683706402778625, 2.618102550506592, -0.16670577228069305, 0.3804013729095459, -1.4013309478759766, -0.47145509719848633, -0.28634190559387207, 0.5540010929107666, 0.25600966811180115, -0.5073682069778442, 1.4939535856246948, -0.4595763087272644, -1.076821208000183, -1.1243096590042114, 1.3560081720352173, 0.9940376281738281, 0.28643208742141724, 0.8302449584007263, -0.48063868284225464, 2.812230348587036, 1.0767086744308472, 0.5261123776435852, 1.5729702711105347, 0.15130789577960968, -0.05293894559144974, -0.3136821985244751, 1.0263409614562988, -1.568721890449524, 0.31854793429374695, 1.0564390420913696, -2.6375741958618164, 1.0530028343200684, 0.42302367091178894, 0.601559042930603, 1.024227261543274, 0.15870234370231628, -0.15964829921722412, -1.0993280410766602, -0.5148447751998901, 0.5956153273582458, -1.1520034074783325, -0.48080089688301086, 0.7810564637184143, 0.47203588485717773, 0.5946969389915466, 0.7848917841911316, 1.0263725519180298, 0.43014219403266907, 0.16672366857528687, 0.35492393374443054, -0.19430460035800934, -1.7985907793045044, -0.23718996345996857, -0.6557286977767944, 1.2442630529403687, 0.6804183721542358, -0.1348978728055954, 0.33941131830215454, 2.321012020111084, -0.7185750007629395, -0.4688454568386078, 0.3827441930770874, -1.8319567441940308, -0.13827578723430634, -1.6749097108840942, -0.6927682757377625, 0.38532140851020813, 0.3410973846912384, 0.016012437641620636, 1.0763580799102783, -0.77609783411026, -0.4571475386619568, -0.13132880628108978, -1.7394832372665405, 0.2684367299079895, -0.636448860168457, -0.7201096415519714, 0.7938665151596069, 0.15598495304584503, -1.0923950672149658, -2.276132583618164, -1.728055715560913, 0.29127761721611023, 0.7103075385093689, -1.4874696731567383, 0.6257145404815674, -0.9507864117622375, -0.9616245031356812, -2.0106911659240723, -0.8165640234947205, -0.18762701749801636, 0.7033076882362366, -1.9968788623809814, 0.15770551562309265, -0.563668429851532, -1.5379945039749146, -2.1457295417785645, -0.24097014963626862, -0.5706632733345032, -1.0159921646118164, -0.6175767183303833, -1.3862653970718384, 1.8099420070648193, 0.828976035118103, -0.27103880047798157, 0.7681310772895813, 0.7804943323135376, 1.8604804277420044, 0.13274076581001282, -1.5567402839660645, 0.9751978516578674, 0.47245511412620544, -0.5126250982284546, -0.8244295120239258, -0.9018712043762207, -0.12733663618564606, 0.7059178948402405, 0.7240997552871704, 0.7760241627693176, 0.872551679611206, 0.7642993330955505, -0.9923316836357117, -0.6352463364601135, 0.49722179770469666, 0.4174002408981323, -1.487766981124878, -2.5632731914520264, 0.13687622547149658, 0.25558164715766907, -0.16584007441997528, -0.2165929228067398, 0.6860622763633728, 0.9753555655479431, -0.9793473482131958, -1.252819299697876, 0.15137992799282074, 2.2547547817230225, -0.06662463396787643, -0.6064773797988892, 0.826451301574707, -0.7020949125289917, -0.7594013214111328, -2.2302403450012207, -0.27239930629730225, -0.9297943115234375, 1.7420629262924194, -0.5786829590797424, -0.5437029004096985, 1.7025078535079956, 0.3106096088886261, 0.8810818791389465, -1.5391020774841309, 0.8609084486961365, -1.3610999584197998, -0.8967010378837585, -0.7577050924301147, -0.12482014298439026, -0.40037864446640015, 1.440822720527649, -1.4229093790054321, -0.6845113635063171, -1.2660094499588013, 1.3441859483718872, -1.2105202674865723, 0.41853660345077515, -0.14841532707214355, 0.28778573870658875, -0.9745014905929565, -2.378633737564087, -1.4014972448349, 0.1396556943655014, 2.1225485801696777, -2.9242653846740723, -0.5915486812591553, -0.07386036962270737, 0.4389009177684784, -0.9346767067909241, -1.2228256464004517, 0.8456422686576843, 1.337988257408142, -0.45536690950393677, -0.516254186630249, 1.3460530042648315, -2.145735025405884, 0.34856587648391724, -0.726455569267273, -0.1230076253414154, 0.6681622266769409, 0.42182061076164246, 0.6742087602615356, 0.1367846429347992, -0.131814107298851, -1.4200024604797363, -0.07289164513349533, -0.32428407669067383, -1.0537712574005127, -0.6337041258811951, 0.8636873960494995, -1.5576080083847046, 0.31650233268737793, -0.9592663645744324, 0.2551761865615845, -0.41934725642204285, 1.2190768718719482, -0.7407196760177612, 0.8218457698822021, 0.9276607632637024, -1.0527656078338623, 0.4326711595058441, 0.5818279385566711, 0.39046260714530945, 0.2550213932991028, 1.7150975465774536, -0.6503527164459229, -1.7608519792556763, -0.2563974857330322, -0.9802115559577942, 0.24952389299869537, -0.6138745546340942, -1.4566738605499268, 0.8483801484107971, 0.9651239514350891, 1.0416760444641113, 0.5522339940071106, -0.11412522941827774, 0.15605877339839935, -0.8714723587036133, 1.0800398588180542, -0.4447520673274994, 0.29846787452697754, 0.35927027463912964, -0.5527591705322266, 0.2434817999601364, 0.9188376665115356, 0.14038142561912537, 0.5796135663986206, 0.3609030246734619, -1.3471362590789795, -1.4078513383865356, 1.3056199550628662, -1.7256195545196533, -0.49812138080596924, -0.06412693858146667, 1.3053771257400513, 0.8021812438964844, 0.6870512366294861, 1.4150216579437256, -0.175072580575943, -0.28356143832206726, -0.417178213596344, -0.5504986643791199, 0.7394586205482483, -1.4713259935379028, -0.5744125247001648, 0.24767562747001648, 1.7091279029846191, 0.8226056098937988, 2.283857822418213, 0.5051842927932739, -0.7581040859222412, 1.680701494216919, -2.5920345783233643, 0.720051109790802, -0.25425663590431213, -1.2770657539367676, -0.3989187777042389, -0.006266476586461067, -0.3086149990558624, -1.5389963388442993, 0.8837817311286926, -0.5194011926651001, -0.43678465485572815, -0.01033856626600027, -2.8784806728363037, 0.9060848355293274, 0.8816530704498291, 1.374835729598999, -0.5572721362113953, 0.18766745924949646, -0.4639548361301422, -0.4015951156616211, -0.4793417453765869, -0.46881553530693054, -0.3337680399417877, -1.2981864213943481, -1.1679490804672241, -1.694158911705017, -0.9769124388694763, 0.029861314222216606, -2.127359628677368, -0.055068910121917725, 2.8839216232299805, 0.7839650511741638, 2.099700927734375, 0.3591824173927307, 0.4071277976036072, 0.8686912655830383, -0.10884305834770203, 0.8170775175094604, -0.60093092918396, 0.8074210286140442, 0.370829313993454, 0.6649641990661621, -0.31644922494888306, -1.5219357013702393, -0.19591595232486725, 0.363190621137619, -1.193273663520813, -1.4596511125564575, 0.42826876044273376, 1.0377461910247803, 0.058570556342601776, -1.2718030214309692, -1.4073293209075928, -0.6349765658378601, -0.59161376953125, -0.08540903776884079, 0.5384939908981323, -0.22367167472839355, -0.32871899008750916, -1.1814167499542236, -1.0602916479110718, 0.7658786773681641, -0.9468005895614624, -0.18543802201747894, -0.4239533841609955, 1.383642315864563, 0.3184782862663269, -0.33449169993400574, 0.34125298261642456, -1.334794521331787, 0.918239414691925, 0.3185485899448395, -2.380361318588257, -0.1034105196595192, -1.272939682006836, 0.26044806838035583, -0.46729201078414917, -0.26392069458961487, 1.5977932214736938, -1.4527214765548706, 1.249732494354248, -2.014394521713257, -1.269775152206421, 0.9581632018089294, -0.9580957889556885, 0.7458135485649109, 0.09641949832439423, 0.07905317842960358, 0.8636633157730103, -0.038069892674684525, -0.6426193118095398, -0.9772854447364807, -0.5673373341560364, -1.030914306640625, -0.751715362071991, -0.7378554344177246, -2.797543525695801, -0.7928586602210999, 0.22274838387966156, 0.06996698677539825, -1.4111230373382568, 1.7965244054794312, -0.8180330395698547, -1.2996735572814941, -0.9349042177200317, -1.3492640256881714, 0.9646281003952026, -0.019589746370911598, -0.7099536061286926, -0.6057838797569275, 0.010274379514157772, 0.2747235596179962, 0.20841048657894135, 1.7620747089385986, 0.5110543370246887, 0.798588752746582, -1.1851742267608643, -0.4024416208267212, -0.43472644686698914, 1.2827047109603882, 1.2142088413238525, 0.6487796306610107, -0.5971261858940125, 2.724574089050293, 1.551438331604004, 1.2612204551696777, 0.5335363149642944, -0.9483069181442261, 0.9742180109024048, 0.3078795075416565, 0.6047390699386597, 0.8592844605445862, -0.3338186740875244, -0.30999845266342163, 0.11718049645423889, 0.1764858067035675, -0.5021570920944214, 0.3569936454296112, -2.1394577026367188, 0.22477594017982483, 0.6940218210220337, 0.5073581337928772, -0.8549742698669434, 0.6577489376068115, -0.5402535796165466, -0.06803485006093979, 0.7258880138397217, 0.7910420894622803, -0.911168098449707, -0.39904242753982544, -0.7269548773765564, -0.170254647731781, 0.6630707383155823, 0.28268593549728394, -1.0683704614639282, -1.0779590606689453, -1.506867527961731, -1.2317808866500854, 1.4498271942138672, -1.0224353075027466, 0.9789549708366394, -1.23261296749115, -0.010498183779418468, -0.09091385453939438, 0.054005686193704605, 1.2852699756622314, -1.695020079612732, -0.857363224029541, 0.3732418715953827, -0.9484161138534546, -1.0435117483139038, -0.7381195425987244, -2.821124792098999, -1.6110225915908813, -0.3185982406139374, -0.9825127124786377, 0.5529963970184326, 0.0016102457884699106, -0.49720025062561035, 0.30266839265823364, -0.1324785053730011, -1.0782538652420044, -0.5213642716407776, -0.941233217716217, -0.7827333807945251, -1.5468884706497192, -0.5222525000572205, 0.6766571998596191, -1.1606535911560059, 0.14217370748519897, 0.49381473660469055, 0.46271777153015137, -1.9379156827926636, -0.8079072833061218, -0.5715270638465881, -0.28834235668182373, -0.7756818532943726, 1.1841061115264893, 0.35232192277908325, 0.22769130766391754, 0.8092270493507385, -0.16917704045772552, -1.4834991693496704, -0.20107322931289673, 0.38642844557762146, 0.3443818986415863, -0.40254640579223633, -1.061340093612671, -0.0568043515086174, 2.2010085582733154, 0.1648818403482437, 0.4522547721862793, -1.9686614274978638, 0.392001748085022, -1.672608494758606, 2.3397083282470703, 0.0957583487033844, -2.9129979610443115, 2.1327366828918457, -0.8601732850074768, 1.2470128536224365, 0.4463440477848053, -0.8997078537940979, -0.8399280309677124, 0.580358624458313, 0.6831823587417603, 0.042586177587509155, -1.141150951385498, 0.6273532509803772, 1.846173644065857, 0.5671026110649109, 0.5899649262428284, 0.8646584153175354, 0.4933799207210541, 0.7236897349357605, 0.7682487368583679, -0.7243089079856873, 0.3692346215248108, -1.156453013420105, -0.0315093956887722, 0.8981142640113831, 0.2972533702850342, -0.4093490242958069, 1.417554259300232, -0.5899831056594849, -0.8347434997558594, 0.14719745516777039, -3.2352073192596436, 0.49001869559288025, 0.39933791756629944, 3.0371320247650146, -0.3163926899433136, -0.12366650253534317, 1.0728623867034912, 1.0622680187225342, 0.43045273423194885, -0.9100589156150818, 1.1200467348098755, 0.2633497714996338, -2.4219775199890137, 0.2994188964366913, -1.4036957025527954, 0.17216812074184418, -0.14276883006095886, 0.25611478090286255, -0.26413238048553467, -2.5794947147369385, -0.25922128558158875, 1.9058443307876587, 0.49299269914627075, 0.8521609902381897, 2.3404529094696045, 0.015454242005944252, 0.07856807112693787, 0.060241129249334335, -0.8168174624443054, -0.911060631275177, 1.7687681913375854, 1.0220175981521606, -0.9591729044914246, 0.6344620585441589, -0.14755937457084656, 0.1953827291727066, -1.1087491512298584, 1.478856086730957, 0.36898303031921387, -1.4454679489135742, 1.0752239227294922, -1.1749577522277832, 0.9022159576416016, 0.46170127391815186, 0.11428917944431305, -0.5377131104469299, -0.596580982208252, -1.0794686079025269, -2.2013723850250244, 0.4756227731704712, -0.8773523569107056, -0.5694226622581482, 1.7148035764694214, 1.251874327659607, 1.7934043407440186, -0.6619144678115845, -0.46324285864830017, 0.2525438368320465, 0.6833788752555847, 0.8862048387527466, 1.1434441804885864, 0.29400792717933655, 0.2538784146308899, -0.8962326645851135, -1.152943730354309, 0.5380433797836304, -0.5182387232780457, 1.4214962720870972, -0.7701751589775085, -0.790489137172699, 2.0571019649505615, -0.735802948474884, 0.036875076591968536, 0.44084659218788147, 1.0196444988250732, 0.3767589032649994, 0.11838511377573013, -1.7380731105804443, 0.39727547764778137, -0.8100159168243408, -0.7326290011405945, -0.4968615770339966, -1.6381804943084717, -2.125910758972168, 0.4979589581489563, -0.5369608998298645, -0.8672786951065063, 0.5679389238357544, -0.8265366554260254, 0.9458858966827393, 0.49461454153060913, -0.2935580611228943, 2.4398627281188965, -0.17161253094673157, 0.40430039167404175, -0.6994573473930359, -0.2282319962978363, 1.0405833721160889, 2.012690544128418, 1.2775815725326538, 0.2997909486293793, -0.07456454634666443, -0.42414942383766174, 0.6586674451828003, 0.6458796262741089, -0.6259760856628418, -0.421726256608963, -0.8638187050819397, 0.6301177740097046, -0.15416274964809418, -0.6474224328994751, -0.24123725295066833, 0.16634930670261383, 0.7686172127723694, -0.591973602771759, 0.7236257791519165, -1.6498185396194458, -0.42827361822128296, -1.263737678527832, -0.739579439163208, -0.6119746565818787, 0.3624778687953949, -1.3688520193099976, -0.6957159042358398, 0.46391138434410095, 0.2401653379201889, -0.30653083324432373, 1.3711897134780884, -0.6833166480064392, -1.0747743844985962, -1.69595468044281, 0.11094354838132858, 1.1188958883285522, -0.37646839022636414, 0.8453195095062256, -0.9982912540435791, -0.488947331905365, -0.44381001591682434, 1.5479165315628052, -1.208591103553772, -0.5527505278587341, -0.26403388381004333, 0.03716982901096344, -1.7714332342147827, 0.13970953226089478, 0.8072026968002319, -0.25345492362976074, -0.4272380769252777, 0.5693063735961914, -1.6551296710968018, -0.5794966816902161, 0.8042528033256531, -0.2885406017303467, -0.8488891124725342, 1.5711878538131714, 0.2782655358314514, -0.08844638615846634, -0.2628319561481476, -0.0058534941636025906, -0.19800283014774323, -0.10546307265758514, 0.03893895819783211, -1.110260248184204, 1.2638497352600098, -0.831925630569458, 0.10360697656869888, -0.23809073865413666, 2.845670700073242, 0.4888264238834381, 2.600691318511963, -0.31646928191185, -0.23079362511634827, 0.07862552255392075, -0.20693287253379822, -1.2243106365203857, 0.9661599397659302, 0.8527122735977173, -1.545272707939148, 1.2180753946304321, -1.0061906576156616, 2.037813186645508, 1.3680657148361206, 0.9421994686126709, 0.1168651208281517, -0.4458339214324951, 0.10694856196641922, -0.6929542422294617, -1.3296616077423096, 0.7544968724250793, 0.7677549719810486, 0.09458094835281372, -0.33068957924842834, 0.5601425766944885, 0.7779651880264282, -0.7382149696350098, -0.2883835732936859, 1.6977753639221191, 0.6349723935127258, 0.31220144033432007, 0.6079553365707397, 1.507621169090271, -1.018929123878479, -0.34116119146347046, 0.6598970293998718, 0.3966589868068695, -0.26166898012161255, 0.47764018177986145, 0.33669835329055786, -1.0662504434585571, 0.3001194894313812, 0.23828363418579102, 1.5161809921264648, 1.6086316108703613, -0.44896844029426575, -2.9065449237823486, 0.7185107469558716, -0.12682394683361053, -1.3293100595474243, 0.8453521132469177, 1.1208165884017944, -1.4254602193832397, 0.7974198460578918, -1.307660460472107, -0.2701152563095093, 0.6046187877655029, -1.225036382675171, -1.7551782131195068, 0.042751215398311615, 0.6443427801132202, -0.7650392055511475, 0.3816085159778595, -0.936621367931366, -0.48250439763069153, -0.6855656504631042, 0.7076816558837891, -0.7877911925315857, 2.2964258193969727, -1.6587028503417969, -0.6320942044258118, 2.414130687713623, 0.1724022477865219, -0.5858909487724304, -1.2464009523391724, 1.1287035942077637, 0.32600343227386475, -1.1691176891326904, -0.2109234780073166, -0.38248512148857117, -2.040384292602539, -0.1556018441915512, -0.2892104685306549, -1.094449520111084, -0.4896920621395111, 0.4705192744731903, 0.2654329836368561, -0.12717536091804504, -0.9891857504844666, 0.06514424830675125, 1.4634305238723755, -1.3861333131790161, -0.27111291885375977, 1.2280397415161133, -0.007978313602507114, -1.048330307006836, -1.6554771661758423, 0.8882685303688049, 0.006126330234110355, 0.5973716378211975, 0.3018430471420288, 0.1251288205385208, 0.176604762673378, -1.0301514863967896, 0.954025149345398, -1.0092597007751465, 0.7343965172767639, -0.35590389370918274, -0.25150346755981445, -0.6259257197380066, -0.06687863916158676, -0.40679508447647095, -1.459153413772583, -0.10519778728485107, 0.14758381247520447, -0.3772335946559906, 1.3903261423110962, 0.5754764676094055, 0.00562158552929759, -0.05535302683711052, 0.15489812195301056, 0.3928643763065338, 0.7710775136947632, 0.8871468305587769, 1.3693370819091797, 0.08672569692134857, 0.4790096580982208, 1.1804876327514648, 0.23197785019874573, 1.289673924446106, 0.5122165083885193, -1.1763445138931274, -0.29895907640457153, 0.689532458782196, 2.0081825256347656, 0.32455408573150635, -0.588840126991272, -0.9415913820266724, -1.1033231019973755, -0.6229445338249207, 0.4878491759300232, 0.9645808339118958, -0.10849934071302414, 2.6801538467407227, -0.5794777870178223, 1.104135513305664, 1.3826407194137573, -0.4867445230484009, -0.4379972517490387, -0.24147413671016693, 0.43193408846855164, -0.8088673949241638, 2.1851587295532227, 0.22964376211166382, -0.8398864269256592, -0.6043815016746521, -1.0637766122817993, 0.3183489739894867, 0.8209386467933655, 0.34980452060699463, 2.0386643409729004, 0.1482737958431244, 0.9009786248207092, -1.350605845451355, -0.8901135325431824, -0.6771623492240906, 0.3233495354652405, -0.7324617505073547, -0.09766936302185059, 1.5181398391723633, -0.3503037095069885, -0.6190919280052185, 0.7404507398605347, 0.15897300839424133, 1.332410216331482, -1.4006248712539673, 0.11294449865818024, -0.07244236767292023, -2.0856010913848877, -0.7598053216934204, -0.3735625743865967, 0.6359889507293701, -0.7320867776870728, -0.21223950386047363, -0.05188822001218796, 1.9647417068481445, -0.8583052158355713, 0.7480497360229492, -0.49746930599212646, 0.1841914802789688, 0.541335940361023, -1.2411845922470093, -0.0022704857401549816, -1.3806089162826538, -0.31529244780540466, -2.517101287841797, -0.8444409370422363, -1.182770848274231, 0.33747628331184387, 0.41466936469078064, 0.1379540115594864, 0.10155615955591202, 0.5048548579216003, -0.49171996116638184, -1.1218323707580566, -0.7664825320243835, 0.11857764422893524, 0.5398575067520142, -0.7595513463020325, 0.9459041357040405, -0.49230754375457764, 0.7187876105308533, -0.5307879447937012, -0.443440705537796, -0.08315379917621613, 0.16856195032596588, 1.0120426416397095, 1.0823208093643188, 0.05658232793211937, -0.6895769834518433, -0.6800677180290222, -0.6993192434310913, 0.6166263818740845, -0.5409435033798218, 0.36506444215774536, 0.1259794533252716, -0.09864996373653412, -1.1483429670333862, -0.5217491984367371, 0.17593328654766083, 0.23757553100585938, -1.3467035293579102, -0.43790408968925476, -0.34959957003593445, 1.3856688737869263, 0.6001760959625244, 0.5808554887771606, -0.3123648762702942, 0.16937321424484253, -1.77892005443573, 1.2572680711746216, -2.9723541736602783, -0.09653189033269882, 1.3198306560516357, 0.9391024112701416, -2.217277765274048, -0.9273999929428101, 0.3728158473968506, -1.2802786827087402, 0.8240771293640137, -0.5824450254440308, 1.4643888473510742, 1.043453335762024, -2.0551791191101074, 0.4785960018634796, -0.8391604423522949, 0.8977673649787903, 0.5851826071739197, 0.4319823086261749, 0.6940550208091736, -0.24367491900920868, 1.0955675840377808, -0.6760668158531189, -0.3848304748535156, -0.539565920829773, -1.4013720750808716, -0.2954604923725128, 0.3278217315673828, -1.1146678924560547, 0.6415668725967407, -1.0921106338500977, -0.6399506330490112, 1.9640926122665405, -0.6886829733848572, 0.6720223426818848, -0.5599426031112671, -1.2399557828903198, -1.1681005954742432, -1.40477454662323, -2.8313491344451904, 0.10943396389484406, 2.1172893047332764, 0.6876830458641052, -0.5228527188301086, 1.0571255683898926, 2.0150957107543945, -0.4585511386394501, -0.451708048582077, -0.05525320768356323, -1.7091606855392456, 0.49393585324287415, 1.8991793394088745, -0.6062408685684204, -1.1244540214538574, -0.18663564324378967, 1.3457272052764893, 1.4019722938537598, -0.5153998732566833, -1.4482364654541016, -1.5420023202896118, 0.3575170040130615, 1.7102726697921753, -1.449864149093628, 0.45966607332229614, 1.4687601327896118, 0.3645305037498474, -0.8849329948425293, 0.23661814630031586, 0.5419425964355469, -0.35143232345581055, 2.2320358753204346, 1.0936702489852905, -0.24632984399795532, -0.7575845718383789, 0.2026391625404358, -2.070826292037964, -0.4387137293815613, 2.1065731048583984, 2.2059011459350586, 0.7178494334220886, -0.819753885269165, 1.2725951671600342, -1.0339909791946411, -0.18140889704227448, -1.716590404510498, 1.222119927406311, -0.033774010837078094, -0.07845227420330048, -0.05622626468539238, 0.07878587394952774, 1.4125418663024902, 0.25195708870887756, -1.596065878868103, -0.14484582841396332, 0.2586308717727661, -0.02896304801106453, -0.9635643362998962, 0.2979713976383209, -0.6711677312850952, -1.7023348808288574, -0.36746522784233093, 1.4451968669891357, 0.38658103346824646, 0.330262154340744, 1.242236852645874, 1.1881651878356934, -0.057361725717782974, -0.09064346551895142, -0.24543684720993042, 0.20547601580619812, 1.010282278060913, -1.8443635702133179, -2.1065707206726074, -0.0025616721250116825, -0.5633947849273682, 0.24843266606330872, 0.8438054919242859, 0.7488055229187012, 0.2847873568534851, 0.3879382014274597, 0.948495090007782, 0.7884174585342407, -0.5164598822593689, 1.174075722694397, -0.8345773220062256, -0.20655770599842072, 0.4338463842868805, -0.2903457581996918, -0.4880964457988739, 0.17005294561386108, -1.4149348735809326, 0.9046689867973328, -0.7819421291351318, 0.17738190293312073, -0.1737542301416397, 0.20017367601394653, 1.6377973556518555, 1.2946826219558716, -0.32094162702560425, 0.5152953267097473, -1.1902843713760376, -1.2357553243637085, 1.2604645490646362, -1.0471049547195435, -1.0564560890197754, 1.8764622211456299, 1.2942696809768677, 0.6698389649391174, -2.2654812335968018, -1.3312442302703857, -0.3462929427623749, -0.4036564826965332, 0.08106430619955063, 1.6834739446640015, -2.265406608581543, 0.12906287610530853, -0.39376237988471985, 0.960354745388031, -1.3365159034729004, 0.8031207323074341, 0.05653132125735283, 1.3273952007293701, 0.7926493883132935, 0.907913088798523, -1.7065646648406982, -0.5785971283912659, 1.2124370336532593, 0.04000711813569069, 1.6658263206481934, -0.8427401185035706, 0.9570410847663879, 1.980088233947754, -0.07765804976224899, -1.4172602891921997, -0.0931263118982315, -0.4191974103450775, 1.5838464498519897, -0.09436391294002533, 0.5290522575378418, 2.4956798553466797, -0.8361150622367859, -1.0343157052993774, -0.5431755185127258, -1.1872605085372925, -1.0646847486495972, 0.41305267810821533, -0.7664133906364441, -2.8462882041931152, -0.9373189210891724, 0.20305797457695007, -0.11930873245000839, -0.875358521938324, 1.0737780332565308, -0.17851057648658752, -0.11156493425369263, -0.3896903097629547, 1.8091932535171509, -2.0260143280029297, -0.02111460268497467, -0.7107719779014587, 0.7123652100563049, 1.5753600597381592, -1.080193042755127, 0.38618502020835876, 0.70894855260849, -0.4259302020072937, 0.549803614616394, 0.10690785199403763, 0.23916880786418915, 1.2417685985565186, -1.3795957565307617, 1.0139400959014893, 0.8917924761772156, -0.24883124232292175, -0.5633489489555359, -1.7416081428527832, -0.5329134464263916, 0.9381739497184753, -0.7264024615287781, -1.2736696004867554, 2.2703912258148193, 0.29915449023246765, 0.7966098785400391, 0.1229839026927948, -0.8706026077270508, 0.3310409486293793, -0.28655707836151123, 0.09691658616065979, 0.7734948396682739, -0.8771204352378845, 1.0171667337417603, -0.9007698893547058, -2.04107928276062, 0.44247859716415405, 0.1758924424648285, 0.41575416922569275, -0.21897859871387482, -0.25401774048805237, -0.9343439340591431, -0.5540937185287476, -0.48076769709587097, 1.788273811340332, 0.60675048828125, 0.7933621406555176, 1.51224684715271, -0.5456962585449219, -1.1267977952957153, 1.1860332489013672, 0.34659919142723083, -0.9269375205039978, 0.20466941595077515, -0.8275315761566162, 1.2015986442565918, 0.9614328145980835, 0.1958123743534088, 1.3491517305374146, -0.37041330337524414, 1.9116519689559937, -2.1614327430725098, 0.5745067000389099, -0.08934349566698074, 0.269008070230484, -0.5595985651016235, -0.5987877249717712, 0.3920977711677551, 0.11177817732095718, 1.9104652404785156, 1.4265321493148804, 0.9559528231620789, 0.0218207947909832, 0.4097944498062134, 0.20009207725524902, 0.23794575035572052, -0.8822675347328186, 0.5288810133934021, -1.2399730682373047, -1.1794646978378296, 1.6191260814666748, 0.4780937135219574, 0.3669488728046417, -0.6407109498977661, 0.19450385868549347, -0.11419935524463654, -0.7377593517303467, 0.9449222683906555, -1.1930656433105469, 1.0699563026428223, 0.8876115679740906, 2.6421561241149902, 1.2117680311203003, -0.12166775017976761, -2.2192318439483643, -0.805637538433075, -0.006763359997421503, 0.8726964592933655, 0.77299565076828, 1.4927290678024292, 0.9416152834892273, -0.2198062539100647, -1.2785733938217163, -0.9523764252662659, -0.34310850501060486, -0.30797886848449707, 0.122820183634758, 0.3261171877384186, 1.5981613397598267, -0.20823676884174347, 1.7362180948257446, 0.8909580707550049, 0.41733211278915405, 0.66478431224823, 0.13579069077968597, -1.0953919887542725, -0.15682841837406158, -0.34960994124412537, -0.5627542734146118, 0.47617509961128235, 0.9228905439376831, 1.0106691122055054, 1.0726344585418701, 0.02556954324245453, -0.8461917042732239, 2.154381036758423, 0.6619765162467957, 0.7056267261505127, 0.6171947121620178, -0.48944705724716187, 0.28425124287605286, 1.3600764274597168, -0.4320816397666931, -1.2919152975082397, 0.14789113402366638, -0.07444439828395844, 0.6693370938301086, -1.381531000137329, -0.5810492634773254, -0.7612423896789551, -1.4761302471160889, -0.890609622001648, -2.9209442138671875, -0.9010494351387024, -0.02101796306669712, -0.3795619010925293, 1.134658932685852, -0.718054473400116, -0.6519618034362793, 1.364513635635376, -1.3160121440887451, 0.9980539679527283, 0.596435546875, 1.1088712215423584, 0.48299843072891235, -1.5978459119796753, -1.1795408725738525, 1.3650435209274292, -0.2608925998210907, 0.08516231924295425, 0.5140292644500732, 1.0177358388900757, 0.7160143852233887, -0.13096976280212402, 0.39409497380256653, 0.3650287687778473, 1.3467968702316284, -2.4496355056762695, -0.5667194128036499, 1.4680246114730835, -0.14571717381477356, 0.08712700009346008, -0.9511368870735168, -1.4076765775680542, -1.0513254404067993, -1.0819159746170044, -0.29214856028556824, -1.0557411909103394, -1.350955605506897, -0.8406546115875244, 0.37832432985305786, 0.5372515916824341, 0.42955929040908813, 0.9419291615486145, -1.20416259765625, 0.045707281678915024, -1.6155929565429688, 1.0606440305709839, -0.22962212562561035, 0.4562486708164215, 1.0701185464859009, -0.5404191613197327, -0.46088147163391113, -1.1365350484848022, 0.19100716710090637, 0.0423504114151001, 0.6664992570877075, 0.06033112853765488, 0.2375572770833969, -0.37642309069633484, 0.8191810250282288, -0.665800929069519, -1.493982195854187, -2.162210702896118, -0.5495359301567078, 0.41551703214645386, 1.3258731365203857, -1.0916615724563599, 0.9060361981391907, -0.2668871581554413, 0.7269872426986694, 1.030150055885315, 2.120911121368408, 0.3020508885383606, 0.27856510877609253, -0.23717443645000458, -0.7574004530906677, 0.25417131185531616, 0.956493616104126, 0.8909059166908264, -0.31510382890701294, -0.3369188904762268, -1.0442707538604736, 0.40664827823638916, -1.0434633493423462, -0.9086061120033264, 1.1248443126678467, 0.35886064171791077, -0.49696144461631775, 0.05845147743821144, 2.087932586669922, 0.7999987006187439, 2.0821259021759033, -0.1735343337059021, -0.24926742911338806, -1.9567086696624756, -0.30268731713294983, 1.3575471639633179, 2.34525465965271, 0.18251271545886993, 0.28587785363197327, -1.1277382373809814, 0.5953368544578552, 1.6344695091247559, 1.3284201622009277, -0.46215519309043884, -1.552351951599121, 0.4376634657382965, -1.369539737701416, -0.9313465356826782, 0.6714869141578674, -1.1250267028808594, -0.5152159333229065, 0.07927384972572327, -0.40777745842933655, -0.23862174153327942, 0.8473398685455322, -0.16727690398693085, 0.9136979579925537, 1.2515513896942139, 0.21859288215637207, -0.3912773132324219, -0.7312928438186646, -0.6683813333511353, -0.11554586887359619, 1.0897904634475708, -0.5726845264434814, -2.0718672275543213, 0.21651698648929596, 0.043384842574596405, -0.2654193937778473, 0.5222756266593933, 0.8562633991241455, -0.9518170952796936, 0.23112250864505768, -0.04944650083780289, 0.4280167520046234, -0.05797836557030678, 0.009922735393047333, -2.2957239151000977, 1.4231349229812622, 0.9708558917045593, -0.2259671688079834, 0.12151035666465759, -3.082017421722412, -0.05831389129161835, -1.1412935256958008, 0.6944170594215393, -0.9162632822990417, 0.8652633428573608, 0.2589883506298065, -0.6988380551338196, 0.6099163293838501, -0.7193129658699036, 0.7581340074539185, -0.9361047148704529, 0.586231529712677, -0.1683136224746704, 0.8363695740699768, 1.4084933996200562, -2.088512659072876, 0.033338144421577454, -0.9659944176673889, 1.544281005859375, -2.040706157684326, 0.6020833849906921, -2.8398544788360596, 0.8681776523590088, 0.32385221123695374, -0.14471851289272308, 1.7234183549880981, -0.10099635273218155, 0.5876048803329468, -1.2319436073303223, 0.33606860041618347, 0.9279711842536926, 2.316329002380371, -0.8354883193969727, -0.013946215622127056, 1.7676410675048828, 2.5592072010040283, 0.3211555778980255, 0.5419002771377563, -1.519432783126831, 0.3917873203754425, -1.8487319946289062, -0.8406524062156677, 0.6011472344398499, -0.827251672744751, -1.469594955444336, 2.808234453201294, -1.9339008331298828, 0.9788969159126282, 0.8517370820045471, -0.592060923576355, 0.6843122839927673, -0.4129597544670105, -1.1712943315505981, -0.4454405903816223, -0.2937554717063904, 1.0417083501815796, 0.6636979579925537, -0.8533247113227844, 1.5206074714660645, -1.602118730545044, -2.0094797611236572, -1.4948561191558838, -0.8124644756317139, -0.5270233154296875, -1.1558756828308105, -0.1367948353290558, 0.0924900621175766, 1.4281387329101562, -0.7794581055641174, -0.0529576800763607, -0.8475626111030579, 0.7012460827827454, -0.27484455704689026, 2.431535243988037, -0.37630099058151245, 1.5959006547927856, -0.14400795102119446, -0.8482770919799805, 0.7378479838371277, 1.86789870262146, -0.38168865442276, 1.000709056854248, 1.0965592861175537, 0.005003311205655336, -0.1853901445865631, 0.8985595703125, 1.626607060432434, 0.8542088270187378, 0.45125168561935425, -1.8279778957366943, -0.9806672930717468, -0.7280213236808777, 0.39709681272506714, 0.7168843746185303, -1.1606255769729614, 0.37507277727127075, -0.16245798766613007, -1.189089298248291, -0.3545971214771271, -0.46383538842201233, -0.06412710249423981, 1.9007824659347534, -0.007166064810007811, -0.48985400795936584, 1.022227168083191, 0.3869299292564392, 0.49495792388916016, 0.7101251482963562, 0.7419324517250061, 2.717646837234497, -0.874450147151947, 0.3661382496356964, 0.1953410804271698, -1.0482420921325684, -0.3194519877433777, 1.5695092678070068, 1.331619143486023, -0.34870803356170654, 1.0775971412658691, -0.19045421481132507, 0.106847383081913, -1.425896406173706, 0.5295187830924988, -0.49059030413627625, -0.911962628364563, 0.19237975776195526, -0.818431556224823, 0.27087974548339844, -1.133406162261963, -2.0808966159820557, -0.8516724109649658, 0.4430261254310608, 1.9569506645202637, -1.1579945087432861, -0.18692773580551147, -1.4966819286346436, 0.26636579632759094, 0.35541027784347534, -0.4100346267223358, 0.7989397048950195, 0.4758973717689514, -0.0031253790948539972, -1.3104113340377808, 0.6394712924957275, 0.025552991777658463, -1.2680068016052246, 0.03556302934885025, -0.8115953803062439, -0.8914216160774231, -1.5821748971939087, 0.12060785293579102, 0.6976090669631958, 0.007582544349133968, -0.41331350803375244, 1.1193006038665771, 0.1694844365119934, -1.1919142007827759, -1.3767495155334473, -2.196728467941284, -1.951644778251648, -0.4198201298713684, -1.309903860092163, -0.188599094748497, 0.28073248267173767, 1.2123358249664307, -0.8539647459983826, -0.009205847978591919, 1.4659429788589478, -0.3329252004623413, 0.01730317994952202, 0.3349911570549011, -0.8692951798439026, 0.4278583228588104, -1.0542093515396118, -1.0032234191894531, -0.8870269656181335, -0.44277292490005493, 0.5189881324768066, -1.0361764430999756, -1.0545271635055542, 0.8224533796310425, 0.5117456912994385, -0.004719163756817579, 1.5231651067733765, 0.3413563072681427, 1.160731315612793, 0.9203163981437683, 0.13476039469242096, -0.7487687468528748, 0.10692405700683594, -0.9022389054298401, 1.0388076305389404, 0.8579129576683044, -0.16998331248760223, -2.2680695056915283, 0.00537346675992012, 1.2661186456680298, -1.7969725131988525, 0.4662575125694275, 0.9963884949684143, 0.4020063579082489, 0.002918918849900365, -1.5362194776535034, 2.148049831390381, -0.4667147099971771, -1.4741065502166748, 0.523705005645752, -0.36329352855682373, 1.12677001953125, 0.13034529983997345, -0.4265443682670593, -0.29828697443008423, -0.688245415687561, 0.7519883513450623, 0.3877001106739044, 0.9792909622192383, -1.7721138000488281, 1.146054744720459, 0.031074844300746918, -1.1070760488510132, 0.2756269574165344, -1.449766755104065, 0.6554564833641052, -0.6759552955627441, 0.6759375929832458, 0.298119455575943, -1.3798749446868896, -1.1868816614151, 0.5273094773292542, 0.8659124374389648, -1.304869532585144, -0.2665928900241852, 0.016391823068261147, -1.3466391563415527, 0.8161948323249817, -1.3910969495773315, -0.029340654611587524, -1.489600419998169, -1.5790783166885376, -1.1845988035202026, 1.1142710447311401, -1.4278086423873901, -0.7807195782661438, 0.6545549631118774, -0.2624640464782715, 0.7302544713020325, -1.1255451440811157, 0.597271203994751, -0.6649340391159058, 0.4634772539138794, -1.8193132877349854, -1.3763409852981567, -1.646218180656433, 0.7040266394615173, 0.5263566374778748, 0.36344248056411743, -0.19559457898139954, 1.024385929107666, -0.3018532693386078, -0.4381328821182251, 0.539760947227478, 0.5414870977401733, -0.9163210988044739, -0.5227530002593994, 0.29593244194984436, 0.410274863243103, -0.15871621668338776, 1.1379084587097168, -1.8575388193130493, -0.3852584958076477, 0.7613633871078491, -1.0419307947158813, 1.124851942062378, -0.07828319072723389, 1.6053824424743652, 0.6620485186576843, 0.8335058689117432, -0.1282796710729599, 1.249874234199524, -0.6597754955291748, -0.08360618352890015, 0.19883941113948822, -0.25424182415008545, 0.9485732913017273, 0.8870608806610107, -0.9586066007614136, -0.6114472150802612, -0.9497678875923157, 0.07034097611904144, 0.24338607490062714, 1.105407953262329, -0.5228345394134521, 0.9537882208824158, 0.8369009494781494, 1.1342799663543701, 0.19090260565280914, 0.08705035597085953, -0.5296937823295593, -0.15609128773212433, 0.28966474533081055, -0.17288871109485626, 0.18891312181949615, 1.832564353942871, 1.1498066186904907, -0.44177454710006714, -0.09093137830495834, -0.6807656288146973, -0.28775158524513245, -0.03440993279218674, 0.4304283857345581, 0.7898086905479431, 0.27643275260925293, 0.7563582062721252, -3.1062941551208496, -1.7160649299621582, -1.6926512718200684, 0.9850547909736633, -2.1567189693450928, -0.058371108025312424, -0.02127791754901409, 0.21602730453014374, 0.1999242901802063, -0.3253236413002014, -0.6700926423072815, -2.320676803588867, 2.9715802669525146, 0.20071877539157867, 0.23425091803073883, 0.999626874923706, -0.3621107041835785, -1.6882003545761108, -0.7692108750343323, -1.2276995182037354, -0.04960247874259949, 0.9928114414215088, 0.5034011602401733, 0.8067219853401184, 1.9665253162384033, 0.43397414684295654, 0.27076098322868347, -1.1442021131515503, -0.6600870490074158, 1.5499153137207031, 0.9722643494606018, -0.3960423767566681, -0.26827868819236755, -1.148227572441101, -1.0865236520767212, 0.5425419807434082, 0.1173655316233635, -1.0708643198013306, -0.3409046232700348, -1.103639841079712, 0.5493913292884827], [-1.0111627578735352, 0.9693024158477783, 0.8510223031044006, -0.12128470093011856, -0.529582142829895, 0.3591521680355072, -0.0692896842956543, 0.8659231066703796, -0.3644613027572632, -0.4853536784648895, -0.8328537344932556, -0.9081594347953796, 0.21760016679763794, -1.5140408277511597, -1.3401743173599243, 0.8209918141365051, 0.5071205496788025, 1.5910447835922241, -0.13588398694992065, -1.357351541519165, 1.987230658531189, -0.23319105803966522, 0.7224450707435608, 0.637729823589325, -0.42935484647750854, -1.2577147483825684, -0.06947683542966843, 1.6148030757904053, 0.5184210538864136, 0.660415768623352, -0.16419421136379242, -0.33075422048568726, -0.757617712020874, -0.7039562463760376, -0.9375005960464478, 0.18924643099308014, -2.36879301071167, -1.6735517978668213, 1.4640361070632935, -1.7718706130981445, 0.44967788457870483, -0.45183151960372925, 0.6237534284591675, -0.3893757164478302, -0.40421605110168457, 1.0825998783111572, 0.7533445954322815, -1.2450746297836304, -1.3452914953231812, -1.2564070224761963, -0.7000352740287781, 0.7202042937278748, -0.6197932958602905, 0.3174363076686859, -0.43688148260116577, 0.5546331405639648, -0.33235645294189453, -0.6129517555236816, -0.2484353482723236, 0.029928283765912056, 0.03454405814409256, -0.10943634063005447, 0.5217424035072327, -0.6661840081214905, -0.3358209729194641, 0.5632326006889343, 1.2959996461868286, -0.04840764030814171, -0.20280656218528748, -1.5310200452804565, 2.3156940937042236, 0.3809123933315277, -0.934427797794342, 2.0265817642211914, -0.42697274684906006, -0.23789586126804352, 0.279956579208374, -0.24841707944869995, 0.13232523202896118, -2.8659989833831787, 0.09512944519519806, 0.4293135404586792, 0.14219926297664642, 0.30628305673599243, 0.11041397601366043, 0.8523872494697571, 0.016444265842437744, 1.3836034536361694, 1.032497525215149, -0.8428486585617065, 1.641178846359253, 1.439136028289795, -1.9732574224472046, 2.324492931365967, -0.6364614963531494, 0.8646801114082336, -1.7213826179504395, 0.79388427734375, 0.509231686592102, 1.2519162893295288, 0.26341888308525085, 0.3508414626121521, -1.3246155977249146, 0.22133611142635345, -0.8048732876777649, 0.1637190282344818, -0.6694053411483765, -0.9958463907241821, 0.1986013948917389, -1.394255518913269, 0.10624902695417404, -0.3788088262081146, -1.0830036401748657, -1.769984483718872, -0.05420442298054695, 1.286180019378662, -0.3326355516910553, -1.5651650428771973, -1.2112594842910767, -0.04253511503338814, -0.2613973319530487, -0.462645560503006, 1.6542731523513794, 1.353486180305481, 0.08650059252977371, 1.4974614381790161, -1.3809764385223389, 0.8777216076850891, 0.014571396633982658, -1.5106056928634644, -0.2941194474697113, -0.0668201744556427, -0.9503852128982544, -1.4748984575271606, 1.9716159105300903, 0.20450487732887268, 1.081541895866394, -0.2636489272117615, 0.5181390643119812, 1.884575366973877, -0.5861192941665649, -1.5033267736434937, -0.42182034254074097, -0.8355104327201843, 0.9291942715644836, -2.0032622814178467, 0.47231802344322205, 1.1673942804336548, 0.5975809097290039, 0.2534433603286743, -1.2712516784667969, -0.2370278388261795, 0.3985726237297058, 0.612521231174469, -0.011052196845412254, -1.377466082572937, -0.5011581778526306, 0.3492162227630615, 0.6681756377220154, 1.1486507654190063, 2.1498770713806152, 0.5178519487380981, 0.2659096419811249, -1.0606662034988403, -0.7781994938850403, 2.24617862701416, -0.42922893166542053, -1.235459804534912, -1.041189193725586, -0.05166994407773018, 0.8807124495506287, 0.4584718346595764, -0.20779579877853394, -0.08220051974058151, -0.743477463722229, -1.0573248863220215, 0.7041640877723694, -0.48645901679992676, 0.7458964586257935, 0.8221344947814941, -0.3898201286792755, 0.05311375856399536, 0.603428065776825, 1.386124849319458, 0.027695033699274063, -0.12159882485866547, 0.6881063580513, 0.7151175737380981, -0.16439183056354523, -0.46219635009765625, 0.6956526041030884, 0.5944987535476685, -0.7637628316879272, -1.8370168209075928, 0.7979370951652527, 0.2573024034500122, 0.5582778453826904, -1.8420723676681519, -1.0809723138809204, 1.0603922605514526, -0.2228751927614212, -0.8630024790763855, 0.48337408900260925, 0.10510621219873428, 1.6785876750946045, 0.6148831248283386, 0.3906184732913971, -1.347747564315796, 0.4993497133255005, 0.42550915479660034, 0.06618264317512512, -1.5503336191177368, 0.14265133440494537, 0.8483496308326721, 0.7755804061889648, 0.8642855882644653, 1.6429678201675415, -1.42552649974823, 0.22579196095466614, 0.17008742690086365, 0.9009486436843872, -0.03612677380442619, 0.6327705383300781, -1.3530774116516113, 0.3137458562850952, -1.1420434713363647, 0.08310425281524658, 0.07247189432382584, -0.980863630771637, 1.0135717391967773, -1.280490517616272, -1.9583282470703125, 0.5206118822097778, -0.7428147196769714, -0.41067224740982056, 0.11477410793304443, 0.39832016825675964, 0.1009499579668045, -1.4082708358764648, 1.121480107307434, 1.2253988981246948, -0.7395933270454407, 0.5253318548202515, 2.2812001705169678, -0.19199858605861664, 0.12156263738870621, 0.8562536835670471, 1.4091812372207642, 0.8401017785072327, 0.5958890914916992, -1.1623799800872803, -0.011586732231080532, 0.18229229748249054, -0.513144314289093, -0.6237966418266296, -0.05910429358482361, -0.13483785092830658, -0.3826178014278412, 0.3077560365200043, 0.6816466450691223, 0.9205111265182495, 0.5057114362716675, 0.99416184425354, -0.9878919720649719, 1.4786758422851562, 0.3113076388835907, 1.292747139930725, 1.211185336112976, 1.3155626058578491, -0.935441792011261, -0.8530891537666321, -1.6657257080078125, 1.2126227617263794, -0.21765674650669098, -0.6759782433509827, -0.6471174359321594, -0.021521415561437607, 0.1281799077987671, 1.3972899913787842, -1.1707309484481812, -1.0607273578643799, 0.18262338638305664, -1.1091996431350708, -0.07810718566179276, 0.5915170311927795, 0.07739675045013428, 0.8923891186714172, 0.21745066344738007, 0.2294844537973404, 0.6500885486602783, 0.4131077826023102, -0.023388884961605072, 1.3740779161453247, -1.5153034925460815, -0.30038195848464966, -0.8259772062301636, -2.0311930179595947, -2.4828381538391113, -0.9156588315963745, 0.5875438451766968, 1.5230722427368164, 0.6265742182731628, -1.2218888998031616, -1.6748651266098022, 0.13475079834461212, -0.6107257008552551, 1.3008289337158203, -0.130620077252388, -1.0537141561508179, 1.7411346435546875, 0.5810969471931458, -1.3757787942886353, 0.5118089914321899, 0.5056109428405762, 1.0059775114059448, -0.41010457277297974, -0.012075892649590969, 0.7389407753944397, -0.5906011462211609, -0.8226127028465271, -0.20131976902484894, 1.5732957124710083, 1.2673234939575195, 0.3345148265361786, -0.3601396977901459, -0.5154187679290771, 0.08777057379484177, -0.27700403332710266, 2.4770257472991943, 0.48870232701301575, -0.39587312936782837, 0.9584881663322449, 0.5153796076774597, -0.17364709079265594, -0.03452753275632858, -1.1313505172729492, 0.621738076210022, 0.4906047284603119, 0.15594957768917084, -1.778943419456482, 0.423584908246994, -1.9227904081344604, -0.7060513496398926, -0.7371706366539001, 0.13752982020378113, -1.7657140493392944, 0.5191202759742737, 0.9117372632026672, 0.6483538746833801, -0.870410680770874, 0.1469324678182602, -2.049236297607422, -1.8733237981796265, 2.261859655380249, -1.1516178846359253, -0.21013320982456207, -0.5274852514266968, 0.8082199692726135, 0.7296949028968811, -0.21137163043022156, 1.2090048789978027, 1.8122178316116333, -0.0652759000658989, -0.002229982288554311, 2.5524747371673584, -0.35637637972831726, 0.42004698514938354, 0.9034028649330139, -2.153399705886841, 0.1907539814710617, 0.6735742688179016, 1.0610277652740479, 0.8829101920127869, 1.514348030090332, 0.20629018545150757, 0.46070486307144165, 0.7161380648612976, 0.16588377952575684, 0.1352822333574295, 0.7624515295028687, -0.35700926184654236, -1.4792866706848145, -1.7682768106460571, -0.18328317999839783, -1.4021730422973633, 1.1091692447662354, 0.0002736443711910397, 0.991733729839325, -0.16974161565303802, 1.8586781024932861, 1.4583991765975952, 0.8122854828834534, -0.32666921615600586, -1.526181936264038, 1.4625635147094727, 1.0007352828979492, 0.726485550403595, 1.135777473449707, 0.1827193796634674, 1.1079928874969482, -0.9075592160224915, 0.21037207543849945, 0.5447860956192017, -0.9411994218826294, -0.8330049514770508, 0.16405093669891357, 1.3623682260513306, -0.7640719413757324, 0.7939850091934204, -0.3884072005748749, -1.5137003660202026, 0.11229997128248215, -1.0722277164459229, 0.6236796975135803, -1.3338289260864258, 0.1890442818403244, 0.22615934908390045, 0.5163567066192627, 0.15988394618034363, -1.7058738470077515, 0.7685631513595581, -1.2584331035614014, 0.7348209619522095, 1.6146963834762573, -1.5029048919677734, 0.22475308179855347, -0.0779350996017456, 0.5881132483482361, 0.18152709305286407, 0.9661929607391357, -0.4760432541370392, -0.287503182888031, -1.1669784784317017, 0.5219228863716125, -2.4206860065460205, -0.13746562600135803, 0.9386987090110779, -0.12698110938072205, 0.9404345154762268, -0.40663522481918335, -0.8665279150009155, 0.39334627985954285, -1.2147711515426636, 1.6912344694137573, -0.046115316450595856, -1.3877544403076172, 0.27401989698410034, -0.3832859992980957, 0.47053205966949463, 0.597040057182312, -0.3267059326171875, -0.5493277311325073, -1.7235580682754517, 1.1536636352539062, -0.07247598469257355, 3.438161611557007, -0.4947669804096222, -0.5373557209968567, 0.6376431584358215, 1.0412076711654663, -0.20129704475402832, 0.6760925650596619, -1.4281338453292847, 0.5828684568405151, -1.8748908042907715, -0.5531220436096191, -0.09947235882282257, 0.19864782691001892, -0.4807639718055725, -1.285058617591858, -0.5927484631538391, 0.7402729988098145, -1.2265102863311768, 1.0816614627838135, 0.10743818432092667, -0.7012131214141846, -0.7153869271278381, -0.6052862405776978, 0.8774477243423462, 1.26981782913208, 1.5477488040924072, -0.5495707988739014, -1.667914867401123, -1.8373103141784668, -0.133014515042305, -1.7943860292434692, -0.30217719078063965, -0.5378708243370056, 0.6000940799713135, -0.26335597038269043, 1.1274515390396118, 1.2466293573379517, -0.8438906073570251, 0.0673615112900734, -2.486316204071045, 0.9609346985816956, 1.0128424167633057, 0.6973350644111633, 0.32040777802467346, -0.5595774054527283, -0.24765560030937195, 1.3053886890411377, -0.8554601073265076, -0.06541481614112854, 0.5467439293861389, 2.80145001411438, -0.2978132665157318, -0.0865025594830513, -0.10616181045770645, 0.6859564781188965, 0.3516584038734436, -1.9430691003799438, -0.25535205006599426, -0.3332996070384979, 0.6027941703796387, 0.3394737243652344, -0.6944229006767273, 1.5965912342071533, -0.7294899225234985, 0.2986019253730774, -1.285082459449768, 0.9276700615882874, -0.34773948788642883, -0.038382191210985184, 2.929475784301758, -0.31281065940856934, 0.46041372418403625, -2.560199022293091, -1.2933400869369507, -0.3259902894496918, 0.7950572967529297, 0.15645897388458252, 0.15858308970928192, -0.2617204487323761, 0.38373512029647827, 0.2275037318468094, -0.3431328237056732, 0.7920226454734802, 1.7934223413467407, -1.1448570489883423, -0.5008402466773987, -1.7141979932785034, -1.0167399644851685, -0.27095338702201843, 1.5856313705444336, 0.519184947013855, -0.22661344707012177, 0.1342475414276123, 0.06824381649494171, 1.537398338317871, -0.041317615658044815, 0.3810440003871918, 0.6962107419967651, -0.6404812932014465, 0.09849333018064499, 0.952721118927002, -0.19228976964950562, -1.4909840822219849, -1.114349365234375, -2.0584163665771484, -0.4839690625667572, -0.16130486130714417, -0.566429078578949, 0.6995850801467896, -0.7390155792236328, 1.9567029476165771, 0.32104796171188354, 0.7564658522605896, -0.4300572872161865, -1.5382661819458008, -1.2717032432556152, -1.5973803997039795, -0.7499038577079773, -0.013527538627386093, -0.8426578044891357, 0.33682781457901, 1.5264042615890503, -0.3480398952960968, -1.4957503080368042, 0.6702305674552917, 0.32934048771858215, 0.6163836717605591, -0.4502471387386322, -0.039492134004831314, 0.9304913878440857, 1.1963846683502197, 1.439663052558899, -0.042868472635746, -1.6254569292068481, -1.7019309997558594, 0.08846372365951538, 0.7015323042869568, -1.447960615158081, 0.8361577391624451, -0.1631426066160202, 0.6380334496498108, 0.7664927244186401, 0.1895495504140854, 0.676982581615448, 1.4083515405654907, 0.3432615399360657, 1.3819427490234375, 1.001290202140808, -0.8854997158050537, -0.3425936996936798, 0.34342584013938904, -1.9726555347442627, -0.09406934678554535, 0.9297729134559631, 0.8275001049041748, 0.8079343438148499, 2.4659581184387207, 0.6240583062171936, 0.7991410493850708, 1.018652081489563, -0.19599859416484833, 1.5535272359848022, 2.462057590484619, -1.6673554182052612, 1.1528605222702026, 0.9388920068740845, -1.3846384286880493, 0.3572899401187897, 0.04108187183737755, -0.23911379277706146, -0.43285152316093445, -0.3299301266670227, 0.21876777708530426, 0.13992007076740265, -0.003222822677344084, 0.2059308886528015, -0.3660067915916443, 3.262068748474121, -1.8747962713241577, -0.43602392077445984, 0.14418816566467285, 1.545871615409851, -1.2225210666656494, -1.4577689170837402, -0.6853886246681213, 0.04571286961436272, 1.0145221948623657, 0.04544884338974953, 1.4507700204849243, -0.40861210227012634, 0.3838081657886505, -0.8336170315742493, -0.40890195965766907, 0.10170600563287735, -0.20576030015945435, 0.9691469073295593, -0.2075524777173996, 1.7043968439102173, 0.03111839108169079, 1.1832362413406372, -1.5455490350723267, -0.020643917843699455, -0.5919647812843323, 0.6259220838546753, -0.8816847205162048, -0.29847875237464905, 0.09948546439409256, -0.4867647588253021, 1.3529411554336548, -0.5732558965682983, 1.4570705890655518, 2.4340810775756836, -0.6114142537117004, -0.9678710103034973, -0.5348723530769348, 0.8063795566558838, 0.007832858711481094, 2.4330172538757324, 3.5221354961395264, -1.1189647912979126, -1.3278437852859497, 0.4382116496562958, 1.0055439472198486, 0.9491371512413025, 0.7156227231025696, -0.026288911700248718, -0.36045193672180176, -1.0293983221054077, 0.20760613679885864, 1.0058698654174805, 0.04533339664340019, -1.0660237073898315, 1.2282066345214844, -0.4876531958580017, 0.8748741745948792, -0.758026123046875, 0.49157360196113586, -0.6065247058868408, -0.788174033164978, 0.8972362875938416, 0.7361148595809937, 0.5845565795898438, -0.9228977560997009, 1.2064573764801025, -1.3782752752304077, 0.10329493880271912, 0.8061968088150024, -1.4489598274230957, 0.4641013741493225, -0.1108630821108818, -1.0979081392288208, 1.1246674060821533, -1.0486366748809814, 1.0939699411392212, 0.5438209772109985, -1.226438283920288, -0.665239691734314, 0.9968980550765991, -0.15385864675045013, 0.3482266366481781, -0.8901833295822144, 1.2464934587478638, -0.769212543964386, 0.08892536163330078, -1.278448224067688, 0.5899400115013123, 0.41044527292251587, 0.5412811636924744, -1.013999581336975, 0.6866695284843445, -0.3396654427051544, -1.4740628004074097, -0.6544124484062195, 0.43273264169692993, 0.856306254863739, 0.8475401997566223, 0.8144964575767517, -0.3691444396972656, -0.7235117554664612, 0.6010465025901794, -0.6146679520606995, 0.7689476013183594, 1.0212537050247192, 0.6724826693534851, -0.07878058403730392, 0.04593956470489502, 0.6536420583724976, 0.5208384394645691, -0.27772051095962524, 1.2931468486785889, -0.8225572109222412, -2.2774977684020996, 1.5149657726287842, 0.8737797141075134, 0.3469478487968445, 0.4500925838947296, -0.8725310564041138, -0.872294008731842, -0.9962741732597351, 0.5194725394248962, -0.882419764995575, 0.07876896858215332, -0.06797800958156586, 0.5413588285446167, -0.05341659486293793, 0.9850879907608032, -0.6966941952705383, -1.5838061571121216, -0.052988048642873764, -0.5242848992347717, 1.7618225812911987, 1.2698312997817993, 0.18400581181049347, -0.6353378891944885, -0.8518685698509216, 0.07735318690538406, -1.310662865638733, 1.9124763011932373, 0.8956981301307678, -1.3182151317596436, 0.11653821915388107, -0.3751971423625946, -1.2847188711166382, -0.25876912474632263, 1.1962134838104248, -0.4332210421562195, 1.0048223733901978, -1.5840617418289185, -0.04223640263080597, -1.6608619689941406, 1.095612645149231, -0.1171155571937561, 0.367496132850647, -0.14981091022491455, -0.04320291429758072, -1.264621615409851, -0.9131286144256592, 0.695063591003418, -1.2382177114486694, 2.4624931812286377, 1.0464982986450195, -0.07455084472894669, -2.4471914768218994, 1.2618132829666138, -0.7426242232322693, 0.1529752016067505, -0.9898955225944519, 0.37265557050704956, -1.5032213926315308, -0.05644449219107628, 0.015261334367096424, -0.24868318438529968, -0.18605734407901764, -0.2025143802165985, 0.2661316692829132, 1.270909309387207, -0.7531548142433167, -0.38952985405921936, 0.6970229148864746, 0.420452356338501, -0.041900455951690674, -0.5645142197608948, -0.9245403409004211, 1.2476942539215088, 1.1561626195907593, -0.8547627329826355, 0.4820120334625244, 0.505448579788208, 0.3392726182937622, -0.2982304096221924, -0.5619654059410095, -0.3904411494731903, -0.06439028680324554, -0.10917915403842926, 0.8952025175094604, 0.7253287434577942, 0.582806408405304, 0.05179702490568161, -0.8941420316696167, 0.3430950343608856, 0.10920491814613342, 1.0252907276153564, -0.010430378839373589, 0.3722279667854309, 0.9740423560142517, -2.068962574005127, -1.6397929191589355, -0.06953642517328262, 1.3726301193237305, 0.07181204110383987, 1.2858809232711792, -0.9380562901496887, -0.06549978256225586, -2.3113181591033936, 0.8023688197135925, 0.9970558285713196, -0.8415756225585938, -0.448472261428833, -1.4737694263458252, 1.5590195655822754, 0.24597914516925812, 0.8061004281044006, -1.1354600191116333, 0.3725942075252533, 0.35877302289009094, 1.556466817855835, 0.2521740794181824, -1.638707160949707, -0.3142838180065155, -1.8213422298431396, 0.5633258819580078, -0.8777837157249451, 0.7746285796165466, 1.2090466022491455, -1.604079008102417, -0.8112912178039551, -0.38401439785957336, 0.5615270137786865, -1.6806098222732544, -0.31387263536453247, -1.0698140859603882, 0.41004279255867004, -0.6590476632118225, 0.3904254734516144, 0.19743095338344574, 1.4395203590393066, -0.22399459779262543, -1.3742626905441284, 0.994827389717102, -0.5451394319534302, 0.14950934052467346, -2.163236379623413, 2.0286059379577637, -1.162076711654663, 0.8554478883743286, 0.7716726064682007, -1.4827314615249634, -1.3695271015167236, -0.6427546143531799, -0.5245535373687744, 0.5889788269996643, -0.8021900653839111, -1.2464417219161987, -0.8549840450286865, -0.8448991775512695, -0.18950791656970978, -1.0586298704147339, -1.5691676139831543, -0.47908642888069153, -0.43805378675460815, 1.3674336671829224, -0.614186704158783, -0.03551999107003212, 0.00742215383797884, 1.3790942430496216, 1.77986741065979, 0.2779614329338074, 0.7111968994140625, -1.5983630418777466, 0.27088817954063416, -1.3342090845108032, -0.5591011047363281, -0.802647054195404, 1.964034080505371, -0.4274081587791443, 0.08126436918973923, -0.5696679949760437, 0.15329033136367798, -1.5403945446014404, 0.6028686761856079, -0.2719348669052124, 0.8807033896446228, -0.7860772609710693, -0.5919588804244995, 0.000993376481346786, 0.24208837747573853, -0.48626813292503357, -1.0495418310165405, -0.7248307466506958, -0.27056390047073364, -0.9320037961006165, -0.7986332774162292, -0.364035964012146, 2.0517802238464355, 0.3064510226249695, -0.31637999415397644, -0.0033399565145373344, -0.2171078324317932, 0.03749369457364082, -1.8078781366348267, 0.028024297207593918, 1.2095695734024048, 0.4636605978012085, -0.5133211612701416, -0.8337613344192505, 0.4575779438018799, -0.17709378898143768, -0.7914509177207947, -0.1981263905763626, -0.2287786453962326, 3.162912368774414, -0.9733703136444092, -1.4682068824768066, 1.0553956031799316, 0.45085909962654114, -0.23582902550697327, 0.5566926598548889, -0.5875112414360046, 0.06959614157676697, -1.3931591510772705, -1.6605712175369263, 1.22944176197052, 1.143836259841919, 0.7847606539726257, 0.999191403388977, -0.2950395345687866, -0.9361616373062134, 1.3534961938858032, 0.5491458773612976, -1.3337256908416748, 2.1652162075042725, -1.1340057849884033, 0.33812737464904785, -0.2905939817428589, -0.8503013849258423, -1.240565299987793, 1.106885313987732, 1.0084476470947266, -1.2503665685653687, 0.23746030032634735, 1.0501291751861572, 0.3289705216884613, -0.3972652852535248, 1.8559445142745972, -0.7411230802536011, 1.5651847124099731, 1.1771196126937866, 1.0268974304199219, -0.42921194434165955, 1.198657512664795, -1.799915075302124, 0.3502422869205475, -0.3652898967266083, 0.6650728583335876, -2.03269624710083, -0.5042643547058105, -1.0059897899627686, -1.5504957437515259, -0.28678736090660095, 0.437360018491745, 1.533096432685852, 1.1684679985046387, 0.754697322845459, -0.2666179835796356, 0.6015225052833557, 0.027101721614599228, -0.39330244064331055, 0.618368923664093, 0.4851878583431244, -1.0145443677902222, -1.3709207773208618, -1.507624864578247, 1.0331100225448608, 0.4003404676914215, -0.5181332230567932, 0.060055460780858994, -0.18728184700012207, -0.4865885078907013, -2.7209949493408203, -0.9876098036766052, -0.46424129605293274, -0.8198036551475525, 0.6893333196640015, -0.11924140155315399, -1.296457290649414, 1.017242670059204, -0.7819094061851501, 1.0073391199111938, 0.14396147429943085, -0.2628710865974426, 0.11844534426927567, 0.37823009490966797, -0.28369420766830444, -0.523471474647522, 0.36360132694244385, -0.5768511891365051, 1.952002763748169, -0.23786409199237823, -0.4722050726413727, -0.940544843673706, 2.364642858505249, -0.665070116519928, 0.26765936613082886, -1.4961037635803223, 0.34600603580474854, -0.518832802772522, -0.26623624563217163, 0.06072308123111725, 1.0627909898757935, -0.5287851691246033, -0.3365612030029297, -0.340451180934906, -0.1406869739294052, -1.8551291227340698, 1.7002465724945068, -1.8970673084259033, -0.880882203578949, -0.5468434691429138, 1.4838403463363647, -0.6057531237602234, 0.5177596807479858, -0.2963201403617859, -0.3752669394016266, 0.2660585641860962, 0.4480409622192383, 0.562929093837738, -0.570409893989563, 0.36632487177848816, -1.101923942565918, 0.9142148494720459, 0.167988121509552, 0.15282608568668365, -0.21131755411624908, 0.716206431388855, 0.23586130142211914, 1.0716731548309326, 0.047252118587493896, -1.695310354232788, 0.7821704149246216, -1.4388184547424316, 0.6346423625946045, -0.71629399061203, -0.28659290075302124, -0.35153669118881226, -0.5109362602233887, -0.2937743067741394, -0.2937256097793579, -0.4298316538333893, -1.2976702451705933, 0.9636052250862122, 0.539778470993042, 0.1545742005109787, 1.6954189538955688, 0.023857586085796356, -0.8486699461936951, 0.0171031653881073, 1.311135172843933, 0.47946688532829285, 0.04211915656924248, -0.925435483455658, 0.9042668342590332, -2.0985007286071777, 0.8836970329284668, 1.4205193519592285, 0.758234441280365, -0.9277742505073547, 0.5136677026748657, 0.4255678057670593, -0.3756364583969116, -1.2029329538345337, 0.3009611964225769, 1.5689729452133179, -0.9807246923446655, 0.037681471556425095, -0.32459133863449097, 1.1685036420822144, -1.2852253913879395, -0.06839167326688766, -1.2369835376739502, -0.7377018928527832, 0.3613845705986023, 0.10697104036808014, 0.7629805207252502, 0.9990883469581604, -0.5954136848449707, -1.4186551570892334, 0.8233444094657898, -1.1324834823608398, -0.5076581835746765, 0.9143024682998657, -0.4172428548336029, -0.29588934779167175, -0.6345909237861633, -1.553591251373291, -1.0629369020462036, 0.011350712738931179, 0.5788264274597168, 0.454674631357193, 0.13046546280384064, 0.700313150882721, -0.29031381011009216, -1.8259602785110474, 1.392972707748413, -0.638583242893219, -0.2645427882671356, -0.8151406049728394, 0.567201554775238, 0.49422016739845276, -0.9746678471565247, 0.8864384293556213, 0.13815075159072876, 0.30379554629325867, 0.22106745839118958, 0.5568501949310303, -0.6093975901603699, 1.218889832496643, 0.37976327538490295, 0.8455759286880493, 0.6907516717910767, 1.7123368978500366, -0.7937914729118347, -0.41012147068977356, -1.4674433469772339, -0.4335141181945801, -0.8662290573120117, -1.4311025142669678, 0.38998985290527344, 0.9301869869232178, 1.2553166151046753, -0.8821746110916138, 1.0274391174316406, -1.0299484729766846, -1.84806227684021, 0.855461835861206, -0.25366154313087463, -2.301138401031494, 0.7603530883789062, -0.40683797001838684, 0.6348913908004761, -0.48162969946861267, -1.043289303779602, -0.5549288392066956, -0.06611846387386322, -0.07639658451080322, 0.22351014614105225, 3.0916104316711426, 1.9341861009597778, 0.0674615427851677, -1.0377212762832642, 0.4488317668437958, -0.3776266574859619, 0.6325101852416992, 0.39389073848724365, 1.040925145149231, -0.16819137334823608, -2.3309590816497803, -0.07361992448568344, -1.0231233835220337, -0.1440121978521347, -0.3201548159122467, -0.6943511962890625, -1.8300238847732544, -0.755667507648468, 0.511527419090271, -0.6491703987121582, -0.9101020693778992, -0.32056963443756104, -0.44490712881088257, -1.529203176498413, 0.7014986872673035, 0.12557201087474823, -0.5305872559547424, 0.3036339581012726, 0.7742156982421875, -0.022617056965827942, -0.14624613523483276, 0.205512136220932, 0.23064522445201874, 0.4002690613269806, -0.03146301582455635, 1.305284023284912, -1.38692045211792, 1.5703866481781006, 0.4527619481086731, 1.540305256843567, -0.6146988272666931, -0.47069790959358215, -1.036438226699829, 2.057922124862671, 0.3764045536518097, -0.7906374335289001, 0.9517982602119446, 0.36469119787216187, 0.34695541858673096, 0.22525504231452942, 0.7218161225318909, 1.758795976638794, 0.47505512833595276, -1.3891795873641968, 0.6514508724212646, -0.9006969332695007, 0.10839803516864777, -0.16749455034732819, -0.4639340937137604, -0.787700355052948, -1.0284849405288696, -0.02804068848490715, 0.5401297211647034, 1.9386565685272217, -0.5227824449539185, -2.066127061843872, -0.7350687384605408, -0.005239848513156176, -0.011577041819691658, -3.2886621952056885, -0.47920045256614685, 1.521016001701355, -2.196106433868408, 0.29527413845062256, 1.5317704677581787, 1.693648099899292, 0.7911035418510437, -0.4827585220336914, 1.182541012763977, -2.625121831893921, -0.2828691303730011, 1.0393749475479126, -0.331516295671463, -0.16648447513580322, 1.0684080123901367, -0.4635193645954132, -1.8402312994003296, -0.6335276961326599, 0.015406000427901745, 1.1255378723144531, 0.13472720980644226, -0.6277032494544983, 0.07317902147769928, -0.43890854716300964, -0.4045782685279846, -0.5041588544845581, 1.1469056606292725, 0.011438014917075634, 0.3265071511268616, 1.5257890224456787, -0.007314835209399462, 0.010234806686639786, 0.9451758861541748, -0.7963059544563293, 0.8487716317176819, -0.33229297399520874, -0.9345275163650513, 0.5324326157569885, -0.03329076990485191, -0.6726289987564087, -1.658115029335022, 0.9972690343856812, -0.6596629023551941, 0.6239450573921204, 0.4862135350704193, 0.14989645779132843, 0.31385454535484314, 0.49711641669273376, -0.5646519064903259, -0.8244835734367371, 1.0931872129440308, -1.548380970954895, 0.74785977602005, 0.40773677825927734, -0.5667861104011536, 1.6004526615142822, -1.0870856046676636, -0.49801793694496155, 0.5791234970092773, -0.875188410282135, -0.4389791190624237, -1.8631876707077026, -0.3168785274028778, 1.0863765478134155, 1.2006522417068481, -0.8660925030708313, 0.5548696517944336, 0.20298035442829132, -0.17032857239246368, 0.36582496762275696, -1.5561872720718384, -0.3760473430156708, -0.5895907282829285, 0.4461890459060669, 2.3285071849823, -0.3635373115539551, -0.4831560254096985, 0.07032380253076553, -0.49853670597076416, 0.015002598986029625, -0.03513870760798454, 0.2060675173997879, 1.3873945474624634, 1.3807939291000366, -0.49554160237312317, -1.3419482707977295, -0.6441894769668579, -2.054823637008667, 0.3476593792438507, 1.132339358329773, -0.7252050042152405, -0.2125113308429718, -0.013763325288891792, -1.3305484056472778, 1.980508804321289, 2.0055453777313232, -0.820030152797699, -0.09330850094556808, -0.19967135787010193, -1.1260987520217896, 1.5366363525390625, -1.980614185333252, 0.7221847176551819, -0.142976775765419, -0.5317137241363525, 0.18552559614181519, 1.4648396968841553, 0.35433152318000793, -2.159466028213501, -1.1296807527542114, 0.03327207267284393, -1.6595046520233154, 0.8008217215538025, -0.9727439880371094, -0.6863991618156433, -1.1008415222167969, -0.1670418232679367, -1.2149986028671265, -1.2117441892623901, 1.2246170043945312, -1.4427810907363892, -0.28292417526245117, -1.265687108039856, 1.106568455696106, 1.2918161153793335, -0.01844688504934311, 1.011176347732544, -0.09775569289922714, -0.39780575037002563, -1.110013723373413, 1.8013490438461304, -0.5709518790245056, 0.6589422225952148, -0.3836422562599182, -2.1549203395843506, 0.045725077390670776, -2.3232240676879883, 0.4259794354438782, 0.49013766646385193, -1.7168493270874023, -0.08006203174591064, -1.8555444478988647, 0.7030980587005615, -0.21562829613685608, -0.05622387304902077, 0.5701274275779724, -1.477945327758789, -0.5474446415901184, -0.38004830479621887, 1.5159711837768555, 0.4596555531024933, 0.42240747809410095, 1.4283033609390259, -0.2972157299518585, -0.7707924246788025, 1.1133469343185425, 0.8453725576400757, -1.6795434951782227, -0.5981599688529968, -1.0664198398590088, 1.2564729452133179, -2.453188180923462, -0.8730745315551758, -1.1057120561599731, 1.673687219619751, -0.4539569318294525, 0.16378465294837952, -2.0734236240386963, 1.1149948835372925, -1.3830751180648804, -0.5594391822814941, 1.6705290079116821, -1.5560256242752075, -0.1187346875667572, -0.04029783606529236, 0.5341721773147583, 1.33063805103302, 1.274228572845459, 0.21473409235477448, -1.7895184755325317, -2.133485794067383, -0.5654590129852295, 0.28153014183044434, -0.07553359866142273, 0.18672364950180054, 2.095290184020996, -0.5597311854362488, -0.049692198634147644, -0.09825807809829712, 0.5308091640472412, -0.8149958848953247, 0.11061549186706543, 0.4581422507762909, 1.7267025709152222, 0.9539594650268555, 1.4338172674179077, 0.03392161428928375, 0.25055786967277527, 0.6959612965583801, -0.3685908317565918, 0.6995006203651428, -0.23822936415672302, -0.9916377067565918, 0.3359997272491455, 1.1410276889801025, 0.611910343170166, 0.33394086360931396, -0.7387408018112183, 0.48584067821502686, 0.28796032071113586, 1.1447460651397705, 0.8611014485359192, 0.551609456539154, -1.8096630573272705, 0.887370228767395, 2.0760293006896973, -0.33173367381095886, -0.2078409641981125, -1.3846622705459595, 1.281976342201233, 0.2070913016796112, 0.4700770080089569, 0.39243510365486145, 0.12237882614135742, -1.2517143487930298, 0.6392076015472412, -2.013421058654785, -0.21142953634262085, -1.1865248680114746, 1.2458665370941162, 1.7688980102539062, 0.7537636160850525, -0.01678401604294777, 0.4694095253944397, -1.160846471786499, 1.869641661643982, -0.28151917457580566, -0.949942409992218, 0.8286665678024292, 0.21268704533576965, 0.6183667778968811, 0.43911489844322205, 0.37001290917396545, 0.8006051778793335, 0.2790089547634125, 0.1872093677520752, 1.379447102546692, 2.079259157180786, 0.8297731280326843, 1.409011721611023, 0.20612919330596924, 0.5704096555709839, -1.5631572008132935, -1.129698395729065, 0.25627970695495605, -1.9823799133300781, -2.22640061378479, -0.5278021693229675, 0.4267652630805969, 1.7800328731536865, 0.8893109560012817, -0.7319821119308472, -2.584909200668335, 0.9267480373382568, -0.5326417088508606, -1.4449613094329834, 1.6681638956069946, 1.9324612617492676, 0.21732908487319946, 1.2544283866882324, 0.8199212551116943, -0.48288819193840027, -0.33773186802864075, -0.9063277244567871, -1.2111186981201172, -0.12460899353027344, 0.908211886882782, 0.05596119910478592, 0.6446175575256348, -0.4845900237560272, -0.612184464931488, 0.7566896677017212, 1.0275728702545166, -2.225890874862671, -0.40411996841430664, 0.6198438405990601, 0.4785698652267456, -0.6814125180244446, -0.8816508650779724, 2.110466957092285, -0.0991862490773201, -0.2761651873588562, 0.6771135330200195, 0.3847388029098511, -0.7315743565559387, 0.9526538848876953, -2.3678781986236572, -0.6692360639572144, 0.3224293887615204, 0.08841243386268616, 1.5552709102630615, 0.29305484890937805, -0.3449280858039856, 0.09755299240350723, 1.9322216510772705, 1.5909640789031982, -1.218900203704834, -1.6384631395339966, -0.5379345417022705, 1.1930299997329712, -0.519873321056366, -0.3626231849193573, -0.7873508930206299, -0.6851645708084106, 1.7715874910354614, 1.3937177658081055, 0.5995985269546509, -0.7738349437713623, 0.9831873774528503, 1.0877081155776978, 0.585874617099762, -0.4077441096305847, -1.0495330095291138, -2.476940393447876, 0.28278931975364685, 0.6092445254325867, -1.445313811302185, -1.4951733350753784, 1.703861951828003, -0.01370658166706562, 2.157932758331299, -0.36066102981567383, -1.0379706621170044, 1.10883629322052, -1.1062878370285034, 0.14838184416294098, 0.10357370972633362, -1.2312958240509033, 0.887033224105835, 0.10149183869361877, 1.1423656940460205, -1.141864538192749, -0.3019515872001648, 0.42002880573272705, 0.990845799446106, -0.8579463958740234, 0.061301007866859436, -0.9487544894218445, 0.639258623123169, 0.2303287386894226, -1.0144447088241577, 0.6165217757225037, -0.636988639831543, 3.5105576515197754, -0.7851557731628418, 0.7691000699996948, -0.3029881715774536, -1.5813093185424805, 2.1113083362579346, 0.21057237684726715, 1.1561039686203003, 0.5873657464981079, 0.2336483597755432, 0.03303462266921997, 0.8660749197006226, -1.347116470336914, -0.02358466014266014, 0.688363254070282, -1.0559278726577759, -0.8157649040222168, 0.9412454962730408, 0.2635352313518524, 2.379157304763794, 1.0219427347183228, -1.2238647937774658, -0.780059814453125, 1.0686719417572021, -1.3497967720031738, 0.515032172203064, 0.21749265491962433, -0.5584852695465088, 0.691936731338501, 1.2945188283920288, 0.8025915622711182, -0.42522701621055603, 0.8314852714538574, 0.13907559216022491, 0.15555064380168915, -0.6844916939735413, -0.6266379356384277, -0.6655893921852112, -1.5864497423171997, 1.0265436172485352, 0.07480908185243607, -0.3369181454181671, 0.16638366878032684, 1.7038124799728394, -0.76845383644104, -0.7237035036087036, 0.042781319469213486, 1.484858512878418, -0.18700408935546875, 0.08061685413122177, 0.699307382106781, -0.7870287895202637, -0.7337645888328552, 0.10337535291910172, -0.41062891483306885, -0.48342469334602356, -1.0452332496643066, 1.2101385593414307, 0.009874891489744186, -0.3662945628166199, -1.1768864393234253, 1.1328744888305664, 0.9611161947250366, -0.10991156101226807, 1.821210503578186, -1.5734062194824219, 0.15522120893001556, -2.8952536582946777, -0.2652769386768341, -0.6423403024673462, -0.25285568833351135, 1.704007625579834, -0.25020697712898254, -1.5976991653442383, 0.7246975898742676, 0.46149054169654846, 2.3331151008605957, -0.9021698236465454, -1.3898340463638306, 0.810996413230896, -0.5006344318389893, 0.927424430847168, 1.3337839841842651, -0.5152954459190369, 0.6203621029853821, -1.219914197921753, 0.30108019709587097, 2.4197542667388916, 0.6393251419067383, -0.6275601983070374, 0.5116258859634399, -0.23211558163166046, 0.21500985324382782, -1.0615804195404053, -0.862398624420166, -0.473165363073349, 0.8007737994194031, -0.7409319877624512, 0.8452779650688171, 0.664637565612793, 0.19498896598815918, 1.3352899551391602, 0.5628425478935242, 2.708934783935547, -0.8421491980552673, 0.26352477073669434, 0.30665838718414307, -0.15138408541679382, -0.699135959148407, 0.6091541647911072, -0.8051130771636963, 0.8357974886894226, -1.7226272821426392, -0.45113465189933777, -0.9383808374404907, 0.4184330999851227, 0.5430036187171936, 2.34733510017395, 1.0594661235809326, 0.14574390649795532, 1.3433668613433838, 0.06502854079008102, -0.003975740168243647, 1.4677706956863403, -0.8076131343841553, -2.370436429977417, -0.9118908047676086, -0.13183946907520294, -0.5474419593811035, 0.10036353021860123, -2.486140727996826, -1.1387438774108887, 0.003947516903281212, -1.5291543006896973, -0.45164042711257935, 0.11702455580234528, 0.06402326375246048, -0.41789668798446655, 1.9767358303070068, -1.7695363759994507, 1.7018747329711914, -1.453959584236145, -0.24367420375347137, 0.8089105486869812, 0.1740369200706482, -0.6793196797370911, -1.337348222732544, 0.30550605058670044, -0.6506894826889038, 1.0232868194580078, -0.32136186957359314, 0.9067673683166504, -0.3338918089866638, 0.9685511589050293, 0.012136576697230339, 0.9298919439315796, -1.1188422441482544, -0.49612879753112793, 0.38003087043762207, -0.5230097770690918, -0.46514347195625305, -0.9958738088607788, 0.8373374342918396, 0.642329752445221, 1.2218966484069824, 0.1477409154176712, -0.5021926760673523, -1.6003391742706299, 0.12671677768230438, -0.4841601252555847, 0.9064890146255493, 0.22170254588127136, -1.9293874502182007, -0.5593233108520508, -0.15261578559875488, -1.343498945236206, 0.7852783799171448, -0.35910847783088684, -3.964935302734375, 1.0300354957580566, 0.5159237384796143, -1.4347530603408813, 0.38908225297927856, 1.2492457628250122, -1.2055203914642334, 1.3796515464782715, -0.6242126226425171, -0.9072776436805725, 0.35268014669418335, 1.029764175415039, 0.40987181663513184, -0.5805104970932007, 0.7753736972808838, -0.15746277570724487, 0.09217844903469086, -0.043991561979055405, 0.2218189686536789, 0.23628288507461548, 1.6353946924209595, -1.2660114765167236, -0.5303027629852295, -0.932457447052002, -1.2772183418273926, -0.4641418755054474, -0.6543862223625183, -0.9338104724884033, 1.358275294303894, 0.5487980246543884, 0.7873683571815491, -0.19086280465126038, -0.6652219295501709, -0.8011754751205444, -0.23205804824829102, -0.22810064256191254, 1.0519641637802124, -1.4401012659072876, -1.257498025894165, -0.6224948763847351, 0.5264154672622681, 1.0027414560317993, 0.6277190446853638, -1.4504954814910889, -0.5412759184837341, -0.2962287962436676, 0.553195059299469, -0.21548239886760712, 1.3277264833450317, 0.34153103828430176, -1.4934042692184448, -0.49269360303878784, 1.2695443630218506, -1.1592282056808472, 0.40347152948379517, 1.7117366790771484, 0.631942868232727, -1.8114055395126343, 0.08699911832809448, 0.4502316415309906, 0.5813648104667664, -0.481052428483963, -0.7014814615249634, 0.83306884765625, 0.2227916121482849, 1.280105471611023, 0.6297045946121216, -0.4761998951435089, 0.33308279514312744, 0.9007008075714111, -0.7261009812355042, -0.7426704168319702, 0.04299343004822731, 1.5502897500991821, 0.2631992697715759, -0.47149941325187683, 0.06746078282594681, 1.4500377178192139, -1.2013258934020996, 0.056201137602329254, -0.6521645188331604, 1.4415223598480225, -0.7074175477027893, 0.31307893991470337, -0.6809521913528442, -1.408973217010498, -0.06597103178501129, -0.025906691327691078, 1.1392037868499756, -1.3053886890411377, -0.8500069379806519, 0.03890792280435562, 0.8619795441627502, 1.6450488567352295, 1.410902738571167, 0.5748022198677063, 0.2622567117214203, 0.5734723806381226, -0.9781501293182373, -0.4018881022930145, -1.3037383556365967, 0.5156668424606323, -0.35747185349464417, -0.8541916608810425, -1.4286694526672363, 0.8389777541160583, 0.518051266670227, -0.2025223672389984, -0.728392481803894, 0.3728993833065033, -0.7599536180496216, 0.2680666148662567, 1.1693843603134155, 1.4264335632324219, 1.1950122117996216, -0.1651482880115509, 0.35911330580711365, 1.5311120748519897, 0.15896759927272797, -0.039267413318157196, -1.8745805025100708, -0.3701842129230499, 1.7016843557357788, -1.3362842798233032, 1.992714285850525, -0.264142781496048, 0.26281267404556274, 0.19451627135276794, -0.23255543410778046, 0.22618421912193298, 0.1800520122051239, 1.4349658489227295, -2.3310868740081787, 0.9909703135490417, -1.1283074617385864, -0.7611153721809387, -0.011296000331640244, 0.3760760426521301, -0.9675621390342712, 1.6466352939605713, -1.1354129314422607, -1.4454511404037476, -0.5914753079414368, -1.6838475465774536, -0.11435339599847794, -0.8556537628173828, 0.9508183598518372, 0.20596103370189667, 0.018946725875139236, -1.4548379182815552, -0.22843046486377716, -0.9069712162017822, -0.8928333520889282, 0.960083544254303, 0.858634889125824, 0.03700313717126846, -0.33068418502807617, -0.3042588531970978, -0.6852471828460693, 0.1640745848417282, -0.06039499118924141, 0.002351165981963277, 0.7069372534751892, -1.2721264362335205, -0.08568808436393738, -0.5767292976379395, -0.310668408870697, 1.0212818384170532, -0.5813665390014648, -0.7269049882888794, 0.3282965123653412, 0.584690272808075, 1.7999985218048096, 0.9263023138046265, -0.21678179502487183, 0.7027546763420105, -0.8120810389518738, -0.9489172697067261, 1.0067191123962402, -0.5053708553314209, 0.554183304309845, 0.3232312798500061, -0.7292898297309875, 0.21552711725234985, 0.8956040740013123, -0.5010590553283691, 1.646921992301941, 0.7362731695175171, 1.334354043006897, 1.8511909246444702, 0.4707232117652893, -1.4519295692443848, 0.4393883943557739, 1.250501275062561, 0.22469283640384674, -2.290900945663452, 0.44557762145996094, -0.8578386306762695, -0.9720491766929626, -1.131312608718872, -0.8532360196113586, 1.6864964962005615, 1.7157199382781982, 0.20333316922187805, 0.5018534064292908, -0.00427823094651103, -1.6608690023422241, 1.0502814054489136, -0.3940446078777313, 2.3363661766052246, 0.6367993950843811, 0.7446744441986084, 1.2096027135849, -0.9218640327453613, 0.14245827496051788, 0.9239240288734436, 1.5715601444244385, -0.970992386341095, -1.1826457977294922, 1.670056700706482, -0.46436139941215515, 0.1017855852842331, 0.7968278527259827, -0.18105296790599823, -0.6185651421546936, -0.7317963242530823, -0.5044137239456177, 0.9447305798530579, -0.9638954401016235, 1.1795036792755127, 0.009998648427426815, -0.2433963418006897, -0.5668410062789917, 0.3603126108646393, -0.5205675363540649, 0.08352363109588623, -0.6605520844459534, -1.5846420526504517, 0.918512761592865, 0.29184606671333313, 0.7437609434127808, 0.2425536811351776, -0.2106473594903946, -0.5978714823722839, -0.46051672101020813, 0.3195721507072449, -0.46932706236839294, 1.475050687789917, 0.6449754238128662, -0.574204683303833, -0.966888427734375, 0.31619182229042053, -3.3819420337677, 0.5995987057685852, 1.4436782598495483, 0.8890148401260376, -0.06901519000530243], [1.2878572940826416, 0.7397900819778442, -0.38574790954589844, -0.4933553636074066, -0.8173983693122864, 0.06145818904042244, 1.124852180480957, -0.802730917930603, 2.53666090965271, -0.9164987802505493, -0.2180505096912384, -0.6931676268577576, -0.4528033435344696, 1.1018202304840088, -0.9090909957885742, 0.592008113861084, -0.5222998857498169, 0.9297011494636536, 0.15105505287647247, 1.0002202987670898, -1.0112890005111694, 0.03230317682027817, 1.3963745832443237, 0.6371961236000061, 0.7427567839622498, 0.6061755418777466, -0.47907039523124695, 0.9478709697723389, 0.4576572775840759, -1.3040235042572021, 1.7888420820236206, -0.21444198489189148, 1.3431494235992432, 0.8826461434364319, 0.31853047013282776, -0.15722794830799103, 1.6625781059265137, 0.8303096890449524, 0.442465215921402, -0.4063907861709595, 0.09966187924146652, -0.7535218000411987, -0.4518892467021942, -1.4757550954818726, -0.9767802357673645, 1.1813502311706543, 0.7267623543739319, -2.4624626636505127, 0.18104521930217743, 1.8539867401123047, 2.559988021850586, 0.3648892343044281, -1.3146378993988037, 1.3984898328781128, -1.9299567937850952, 0.9867823123931885, -0.6539869904518127, -0.20070363581180573, -1.0094976425170898, 1.3836642503738403, 0.004201292060315609, -0.9336342215538025, -0.4016849994659424, 2.1345131397247314, 0.1303691565990448, -1.4196369647979736, 0.25718072056770325, -0.555546760559082, 1.4914355278015137, -2.8481783866882324, 0.18904902040958405, 0.7842836976051331, -0.03169974312186241, -1.8980913162231445, -0.8559353947639465, 1.210608959197998, -1.3563565015792847, -1.7135146856307983, -1.495444655418396, -2.186201572418213, -0.3538435101509094, 0.10294171422719955, 0.36093100905418396, -1.2056255340576172, -1.0754144191741943, 0.6758201122283936, -0.3559670150279999, -0.2702956199645996, 1.2120065689086914, -1.0570135116577148, -1.2866332530975342, 0.05958542227745056, 0.004247717093676329, 0.08116623759269714, 0.7458220720291138, -0.8276945352554321, 0.5992555618286133, -1.3717155456542969, 0.9375446438789368, 1.2075177431106567, -0.137348935008049, 0.010012195445597172, -1.2150657176971436, 0.025441506877541542, 0.20862030982971191, 0.1932649165391922, 0.999855637550354, -0.5743291974067688, -0.9987338185310364, 0.6660484671592712, 0.976054847240448, -0.3789418041706085, 0.8304091095924377, 0.3784334659576416, -0.23619164526462555, -1.1774953603744507, 1.8732123374938965, 0.27800774574279785, 0.6685422658920288, -0.8493138551712036, -0.9917768239974976, 1.2883585691452026, 0.4085215926170349, 0.6889174580574036, 0.2222142070531845, -0.19181960821151733, -1.013511300086975, -0.2539535164833069, -0.04392305761575699, 1.107088565826416, 1.4458099603652954, -0.767947256565094, 0.9003006815910339, 0.3062002658843994, -0.07444167137145996, 1.7738020420074463, -0.33653512597084045, -0.10312367975711823, 0.8078873157501221, -0.17587971687316895, 1.7693877220153809, -0.547140896320343, 0.010948716662824154, -0.736014187335968, 0.06686072796583176, 0.9844225645065308, -0.5559855103492737, 0.9229114055633545, 0.015924150124192238, 0.18584825098514557, -0.830569863319397, 0.6122151613235474, -0.7130592465400696, -1.1179180145263672, -1.7592626810073853, 1.893233060836792, 0.04754156619310379, 0.2695614695549011, 1.3831336498260498, -1.0762028694152832, 2.8041481971740723, -0.7714327573776245, -0.6399222016334534, -1.1603341102600098, 0.40426507592201233, 0.645563006401062, 1.384639024734497, 0.2752295434474945, -1.973557710647583, -0.11598890274763107, 1.9827607870101929, -1.0648198127746582, 0.6231571435928345, 0.16039523482322693, 1.2465829849243164, -0.3367772698402405, 0.2836099863052368, 0.3502083718776703, -0.6318018436431885, -0.5661003589630127, -0.38024547696113586, -1.3627625703811646, -0.5407572388648987, 1.0053415298461914, -0.9555997848510742, 0.5628469586372375, 1.192278504371643, 0.5576420426368713, -0.15433816611766815, 1.865461826324463, 0.2890738844871521, 0.28462404012680054, -1.718024730682373, -0.8593006134033203, -0.021006522700190544, 0.9222410917282104, 0.09273673593997955, 0.44700029492378235, 0.5945531725883484, -0.3617246150970459, 1.0233159065246582, -0.5294004678726196, 0.48137596249580383, -0.18653368949890137, -1.8342653512954712, -0.18005229532718658, 1.2633378505706787, 1.0237162113189697, 0.4869873821735382, -0.03557099401950836, -0.3061959445476532, -0.13125093281269073, -2.3822739124298096, -0.5202734470367432, 0.3296763598918915, -0.25583845376968384, -1.0676885843276978, 1.7562438249588013, 0.27429622411727905, -0.40081787109375, -1.2685719728469849, 0.22676336765289307, 0.17532184720039368, -2.83504056930542, 0.44096988439559937, 0.6018937826156616, 1.010722279548645, -2.084529161453247, 0.8105276823043823, 0.0707339495420456, 0.2864936590194702, -1.5421000719070435, -0.2098769098520279, 1.0877060890197754, 0.08495673537254333, 0.34583020210266113, -0.060607366263866425, -0.40773651003837585, 2.0951709747314453, 0.015339898876845837, 0.3660590350627899, 0.6482712626457214, 1.5585081577301025, 0.3155825734138489, 0.40446582436561584, -0.8586690425872803, 1.1977125406265259, 1.556600570678711, 0.11521785706281662, -0.5540809631347656, 1.4623818397521973, 2.2063565254211426, -0.4668107330799103, -0.5552982091903687, -0.09151386469602585, 1.4089127779006958, -0.27238428592681885, -1.3776112794876099, -0.4952109456062317, 0.3386061191558838, 0.7492195963859558, -1.1618292331695557, -0.369831383228302, -0.7706526517868042, 0.5867173075675964, -1.2604598999023438, 1.656530499458313, -0.26594141125679016, -1.7125511169433594, 0.9045172333717346, -1.6738697290420532, 0.6352507472038269, -0.7689575552940369, -0.2022985816001892, -1.4810842275619507, -0.030356494709849358, 0.6846153140068054, 0.08117153495550156, -0.5184364318847656, -1.2738286256790161, -0.41798534989356995, 0.7537094354629517, 1.235993504524231, -0.36088505387306213, -1.968414306640625, -0.10319037735462189, -2.4004693031311035, 1.3355052471160889, 0.8677561283111572, -1.579241156578064, -0.48137202858924866, -0.2799837589263916, -1.3073744773864746, -0.17966407537460327, -0.3320173919200897, 1.4515576362609863, 0.4030984938144684, -0.07706179469823837, 1.1820820569992065, -0.28474193811416626, 2.247006416320801, 0.7688568234443665, -0.04417038336396217, -0.03821954131126404, 0.6274469494819641, 0.24120379984378815, -1.159149169921875, -1.8583018779754639, -0.35704168677330017, -0.9594693183898926, 1.296610713005066, -0.725616455078125, 1.1768277883529663, -2.4364089965820312, -0.14706772565841675, 1.7922395467758179, -0.36636000871658325, 1.0582269430160522, -0.7832428812980652, 0.8185246586799622, 0.7588746547698975, -0.29271188378334045, -2.3022000789642334, 0.1444743275642395, 2.438645601272583, -0.5599963665008545, 1.8938301801681519, -0.7055606842041016, 1.2705532312393188, 0.4146456718444824, -0.6530964374542236, -0.5637139081954956, -0.16518694162368774, -0.9808428883552551, 1.2204110622406006, -0.17717507481575012, -0.30348873138427734, -0.35712388157844543, -0.07312926650047302, -0.09420803934335709, -0.8007929921150208, 0.27140066027641296, -0.4459925889968872, -1.6290810108184814, -0.03042607009410858, 1.4048653841018677, -1.720724105834961, 0.4297158420085907, 1.1707541942596436, 2.3645691871643066, 0.3026456832885742, 0.43876826763153076, 1.5572001934051514, -0.31008803844451904, -0.19938024878501892, 0.07544398307800293, -1.8127106428146362, -0.07408847659826279, -0.340038925409317, 0.3020879030227661, 0.370721697807312, -0.1157732903957367, -1.339806079864502, -0.3112906813621521, -0.8916317224502563, -1.5417850017547607, 1.5022050142288208, 0.3258267939090729, 0.1518317610025406, -1.5257185697555542, -1.6305478811264038, 0.868080198764801, -0.7337736487388611, 0.5477936267852783, -0.24937495589256287, -0.20731547474861145, -1.0124891996383667, 0.5574538111686707, 0.19421637058258057, -1.3069151639938354, -0.7996999621391296, -0.03915581479668617, 0.6219168305397034, 0.19431743025779724, -2.6385180950164795, -0.42057597637176514, -1.1433407068252563, -1.136691689491272, -0.42493686079978943, 0.047885991632938385, -1.6545631885528564, 1.5003246068954468, -0.643714189529419, 0.7875919342041016, 0.4982551634311676, -1.0318660736083984, 0.10344143211841583, -0.48274415731430054, -0.51634281873703, 1.4866517782211304, -0.6142377257347107, -0.16050630807876587, -0.4539273679256439, -1.060196876525879, 0.39305585622787476, -0.17796440422534943, -1.0106185674667358, 1.0713142156600952, -0.03288170322775841, -1.6404964923858643, 1.0269834995269775, 1.4877876043319702, -2.3256049156188965, -0.09422258287668228, -0.693849503993988, -1.0621697902679443, -0.5653262734413147, 0.8707598447799683, 1.0676543712615967, -0.10155657678842545, 0.6818611025810242, 1.450805902481079, -0.5566156506538391, 0.8608406782150269, -0.7937323451042175, 0.7833046317100525, 1.5784107446670532, -0.5212329030036926, 0.24758130311965942, -0.3570180833339691, 1.1091294288635254, -1.076979637145996, -0.8312744498252869, -1.53180730342865, 0.8485620021820068, 1.7761896848678589, -0.7335233092308044, 0.6190896034240723, 0.34047287702560425, -0.940230131149292, -0.10389616340398788, 0.29473525285720825, -0.766651451587677, -0.5819888710975647, 0.6727205514907837, -1.0682095289230347, 0.5524307489395142, 0.26495087146759033, -1.2933837175369263, -0.042747773230075836, 1.1885865926742554, -0.36784788966178894, -1.4515626430511475, -1.7841238975524902, 0.07465457171201706, -0.30689314007759094, -0.6481090188026428, 0.20016779005527496, 0.2687804698944092, 0.7544567584991455, -0.6196542382240295, 1.117100715637207, -0.9958586096763611, -0.10681577771902084, 0.5003310441970825, -0.22099657356739044, 0.2660995125770569, -0.6860011219978333, 0.152086079120636, 1.5543283224105835, 0.8426415920257568, -1.4170992374420166, -0.8824061751365662, 0.2236412763595581, -0.4904874265193939, -0.47994527220726013, 0.16991789638996124, -0.7331563234329224, -0.9310280680656433, -1.6901377439498901, -1.9921537637710571, 1.0206358432769775, 0.5054357051849365, 1.3612449169158936, 0.9795817732810974, -1.9514936208724976, 1.352893590927124, 0.49642106890678406, -1.251340389251709, 0.3230244517326355, -1.0056705474853516, 0.8047763109207153, -1.7695143222808838, -1.460123062133789, 0.12262240797281265, -1.6396969556808472, -1.58134126663208, 0.10775169730186462, 0.5045952200889587, -1.1017944812774658, 1.3177051544189453, 1.4775680303573608, -0.19345705211162567, 0.36175408959388733, -0.39489588141441345, -0.37893009185791016, 0.4387122392654419, -0.4448210597038269, -0.44066229462623596, -0.38073086738586426, -0.3508836030960083, -0.9007691740989685, -0.12966379523277283, -0.76353919506073, -0.829028844833374, -0.4419354796409607, 1.414704442024231, 0.6616660356521606, 0.2893463671207428, -0.17588388919830322, 0.02941632643342018, -0.3049052953720093, -0.5431827902793884, 0.9016557931900024, 0.8680588603019714, -0.9703556895256042, -1.3430145978927612, 1.3163232803344727, -0.7079740166664124, -1.166433334350586, 0.274898886680603, 1.1021567583084106, 1.2301963567733765, -0.15490995347499847, 0.7552927136421204, -0.9067170023918152, -0.7757717370986938, -1.1490086317062378, 0.7389044165611267, -0.21505282819271088, -0.8441073894500732, 0.12642449140548706, 0.799172580242157, -3.231455087661743, 0.6493858695030212, -0.38205477595329285, 1.3492450714111328, -0.4166678190231323, 0.6726244688034058, 0.3823786973953247, 0.5357081890106201, -0.8144005537033081, 1.5067123174667358, 0.013537219725549221, -0.9392671585083008, 0.003282176563516259, 1.7898749113082886, -0.043297212570905685, 0.48712533712387085, -1.0441282987594604, -0.8690575957298279, -0.6842359304428101, -0.17278869450092316, -0.31594815850257874, 0.4889812469482422, 1.1571513414382935, -0.6800414323806763, 0.6474432945251465, -0.9658300876617432, 0.05533948913216591, 0.009078209288418293, 0.31329378485679626, -1.4450695514678955, 0.5814589858055115, 1.417283296585083, -1.3757082223892212, 2.2046585083007812, -1.0297021865844727, 0.7000421285629272, 0.6005284786224365, -0.8957004547119141, -0.9939841628074646, 1.7771533727645874, 0.8277272582054138, 0.8953818082809448, -0.4909208416938782, 1.7040605545043945, -0.4149337410926819, -0.27313053607940674, 1.2450954914093018, 0.024036984890699387, 1.112720012664795, 0.056587930768728256, 0.6635064482688904, 1.122510552406311, -0.33548644185066223, 0.5407313108444214, 1.2589600086212158, -0.7165853381156921, 1.164698839187622, 1.0396853685379028, 1.589220643043518, 0.4026639461517334, -0.03876955434679985, -0.455505907535553, 0.3131089508533478, -0.9328896403312683, -0.03849116340279579, 0.5065450668334961, 2.215731382369995, 0.16727468371391296, -0.5202328562736511, 1.4083367586135864, -0.3790334463119507, -0.717430591583252, -0.41734829545021057, -1.2200316190719604, -0.14141660928726196, 0.7429415583610535, -1.1036710739135742, -1.9056233167648315, 0.2909919023513794, -0.9367831349372864, 0.8647422790527344, -0.4090050756931305, 2.5893027782440186, 0.9021105170249939, 0.5166666507720947, 0.8270112872123718, 1.7582648992538452, -1.6686748266220093, 0.6253432035446167, -0.5734203457832336, 0.6564645171165466, -0.3737233579158783, -0.38088107109069824, -0.11240340769290924, 0.7257099747657776, 0.7714594006538391, 1.123297929763794, 1.3050110340118408, 0.3590586185455322, -1.663182020187378, -1.3631402254104614, -0.0699634775519371, -2.6754324436187744, -1.293720006942749, -1.358327865600586, -0.22934018075466156, -0.14066408574581146, -2.018343925476074, 1.358951449394226, 0.5629298686981201, 0.6131513118743896, -1.0476024150848389, 0.33052778244018555, -1.0266519784927368, 0.22607429325580597, 0.3081052303314209, 0.19877175986766815, 0.7349947094917297, 0.8053241968154907, -1.2596524953842163, 0.11640358716249466, 0.009362280368804932, 0.3510400056838989, -0.3756307363510132, 1.3584645986557007, -0.004952853079885244, -0.09462455660104752, 0.478623628616333, -0.32849135994911194, -0.18511831760406494, 0.47211700677871704, -1.0858882665634155, 1.4345263242721558, 0.6759294271469116, -0.20138075947761536, -2.3295559883117676, 0.6821045875549316, -1.0713046789169312, 0.5257845520973206, 1.8550901412963867, 2.5991129875183105, 0.9047523140907288, 1.4950370788574219, -0.005225574132055044, -1.51067316532135, 0.6304514408111572, -0.22632698714733124, 0.3871629238128662, 0.6733695864677429, -0.7198203802108765, 0.024542534723877907, -0.1834811270236969, 0.8051250576972961, -1.3751356601715088, 0.8499545454978943, 1.8130450248718262, -1.4271713495254517, -0.2320815771818161, -0.6778505444526672, -0.28122779726982117, -0.7354060411453247, 0.08612474054098129, -0.30686452984809875, 0.7992611527442932, -1.3716992139816284, -1.390673279762268, 0.7238701581954956, -0.21697258949279785, -0.7291109561920166, -1.8028454780578613, 0.03735615313053131, 0.2707541882991791, -2.3130390644073486, -1.2241569757461548, -0.24800118803977966, -0.6454507112503052, -1.1765538454055786, 0.06049615517258644, -0.6731526851654053, -0.29771992564201355, -1.8599612712860107, -0.963291585445404, 0.620750904083252, -1.4672672748565674, 0.8302958607673645, 0.5562095642089844, 1.8070017099380493, -1.1146039962768555, -0.05424606427550316, 1.4615833759307861, -0.026178469881415367, 0.5822963714599609, -0.12193398922681808, -0.10403391718864441, 0.6097651720046997, -0.9217458963394165, 1.6014199256896973, 0.5560726523399353, 0.8510733842849731, -2.4733681678771973, 0.8497109413146973, 0.10463521629571915, 0.9300217628479004, -0.09107812494039536, -0.7670377492904663, -0.43543460965156555, 1.0547254085540771, -1.1953181028366089, 0.1429782211780548, -0.5410887002944946, -0.5282642841339111, 0.35831013321876526, 0.24091941118240356, 0.6317891478538513, 0.7600154876708984, -0.534723162651062, -0.6536523699760437, -1.2463990449905396, -1.9765008687973022, 0.8931125402450562, -0.6085777878761292, -1.5223280191421509, -1.3303394317626953, -0.24424134194850922, 0.5378544330596924, -1.1828136444091797, -1.0405641794204712, 1.7789503335952759, 0.014946992509067059, -1.7152711153030396, 1.090248703956604, -0.19012205302715302, 0.4048665761947632, 1.322028398513794, -0.5511536002159119, -0.05474235862493515, -0.7231065034866333, 0.8840174674987793, -0.07122773677110672, -1.5903958082199097, -0.49919018149375916, 0.36956730484962463, -0.1047174409031868, -0.27679088711738586, 1.021519660949707, -0.21883928775787354, 0.36330342292785645, -1.378349781036377, -1.7156741619110107, -0.9732971787452698, -0.8533383011817932, -1.4802582263946533, -0.17376378178596497, 1.691489338874817, 0.7442912459373474, -0.10269749909639359, -0.52403724193573, -1.8668668270111084, 0.03135977312922478, 0.4652024805545807, -0.9071866869926453, -0.15462152659893036, -1.2209904193878174, -0.634222149848938, 0.3617582619190216, 0.5377695560455322, 0.3334754407405853, -0.16980046033859253, 0.23480570316314697, 0.9658934473991394, 0.1543138325214386, 0.8991681337356567, -0.59552001953125, -0.051342375576496124, -0.38569560647010803, 1.3069441318511963, 1.0385844707489014, 0.639139711856842, -0.8203955888748169, -0.416213721036911, 0.8485949039459229, 0.4693436920642853, 0.9419729113578796, 1.0011590719223022, 1.848598837852478, -0.03179236873984337, 0.36657604575157166, 0.2152528166770935, -0.4936424493789673, -0.552948534488678, -0.7465248703956604, 0.03640510141849518, 0.9006057381629944, 1.5193628072738647, 2.5470941066741943, -0.5243517756462097, -0.6185957789421082, 1.881447434425354, -0.9687029123306274, -0.48113518953323364, 1.5368237495422363, 0.9559714198112488, 0.07591221481561661, -0.11211788654327393, 0.9593339562416077, 0.1727408468723297, -2.4435462951660156, 0.6060997247695923, 0.366249144077301, 1.1568082571029663, -0.5444419980049133, -0.8179247975349426, -0.16841943562030792, 1.2217907905578613, -1.501883864402771, 0.027547528967261314, 0.5551932454109192, 0.11765221506357193, -0.7470987439155579, 0.14534255862236023, 0.033679746091365814, -0.18580128252506256, 1.2601062059402466, -0.5026591420173645, -0.5508481860160828, -0.10144948959350586, 1.2208116054534912, -0.1533135026693344, 0.9323348999023438, 0.757332980632782, -1.1963804960250854, -0.00499045941978693, 0.12761442363262177, 0.24373801052570343, -0.4226059913635254, 1.406100869178772, 0.6832616925239563, -1.2508145570755005, -0.9339321851730347, 0.40712031722068787, -0.9246659278869629, 0.24494662880897522, 1.7901182174682617, -1.026130199432373, 0.9602870345115662, -0.6534508466720581, -0.7793000936508179, -0.28320443630218506, -0.37684956192970276, -0.1043921485543251, -0.6771216988563538, 0.5582174062728882, -0.9000309705734253, -0.44456467032432556, 0.42282557487487793, 1.9381951093673706, 0.24054357409477234, 1.0412880182266235, -0.06556295603513718, -0.9391829371452332, -0.4334382712841034, -1.8776477575302124, 0.3195726275444031, -0.06241673231124878, 0.58018958568573, 0.05191759392619133, -0.1362265646457672, -2.005005121231079, 1.0474828481674194, 0.39499804377555847, -1.398483157157898, 0.48493754863739014, 0.4347619116306305, 0.8752217292785645, -1.7771039009094238, -0.04692382365465164, 1.5053396224975586, 0.5049753189086914, -0.9075338244438171, 0.521072506904602, 0.07355232536792755, -0.9306252002716064, -1.7388160228729248, -0.8247509002685547, -1.0348163843154907, 0.4503757655620575, 0.5435706973075867, 0.14081710577011108, -0.48421797156333923, 1.1872529983520508, -1.8692848682403564, -0.7932802438735962, 0.4154503047466278, -0.851294755935669, 1.5411986112594604, -1.9040968418121338, -0.8276681900024414, -0.7527744770050049, 0.9708539247512817, 1.194513201713562, -2.0305848121643066, -1.2292556762695312, 0.38452205061912537, 0.8997934460639954, -0.47154316306114197, 0.16946084797382355, 0.03330666571855545, 0.5183716416358948, -0.269313246011734, -1.2569241523742676, 1.8363114595413208, 0.04466215893626213, -2.187300682067871, 1.3259118795394897, -0.15239369869232178, 1.3967151641845703, -0.5161498785018921, -0.1717481017112732, 1.120621681213379, -0.1946386694908142, -1.1756832599639893, -0.4030168354511261, -0.8373085260391235, 0.613603413105011, -1.2095122337341309, 1.5599963665008545, 0.48885926604270935, 0.10350791364908218, -0.050012048333883286, 0.06543471664190292, 0.37562406063079834, -1.4843910932540894, -0.5387418270111084, -1.2543296813964844, -1.0353281497955322, 0.7506505250930786, -1.4989970922470093, -0.8555880188941956, -1.39120352268219, -1.784110188484192, -0.5068655610084534, 0.6416375637054443, -0.7833137512207031, 0.14023709297180176, -0.9209458827972412, 0.6471987366676331, 0.4391103684902191, -0.4220212399959564, 0.22063569724559784, -0.22299984097480774, 1.049767255783081, -0.8144105076789856, 0.10720399022102356, -0.20063291490077972, 0.3143124580383301, -0.41953620314598083, 1.008041262626648, 0.7340974807739258, -0.9761953949928284, -0.2837679088115692, -0.2946767807006836, -0.40820813179016113, -1.2179404497146606, 0.48884108662605286, -0.8825339674949646, -0.6401360630989075, 0.7123534679412842, 1.6347224712371826, 0.025332750752568245, -0.09375593811273575, -1.5878984928131104, -0.021288203075528145, 0.9976102113723755, -0.6921968460083008, 1.315914511680603, -0.11696388572454453, -0.9752099514007568, -0.8323792219161987, 1.080967664718628, 0.3263814449310303, -0.1756807416677475, 0.1376127302646637, -0.22771388292312622, 0.3825902044773102, -0.6636500358581543, -0.006928527262061834, 0.4899768531322479, -1.391601324081421, 0.5603485107421875, 0.9195801615715027, 0.8265637159347534, 0.24877463281154633, -1.9638142585754395, -1.1604334115982056, 0.8882595896720886, 1.825461745262146, 0.9583565592765808, -1.073901891708374, 0.39069491624832153, -1.3510674238204956, 0.1764056384563446, 0.7635036706924438, 1.6734063625335693, 1.1091678142547607, -0.47288909554481506, -1.6295585632324219, -0.003223707666620612, -0.5235779881477356, -0.15309375524520874, 1.6216391324996948, 1.328945279121399, 0.3723694384098053, -0.5103789567947388, -0.9461119174957275, -0.2099081575870514, -2.016979694366455, -1.3334063291549683, 0.8836466670036316, 3.0729408264160156, 0.9498881697654724, 0.699803352355957, 0.8777077198028564, -0.25851401686668396, 0.18595393002033234, -0.39964544773101807, 0.005422655493021011, -0.2758694589138031, -2.2540626525878906, 1.4949880838394165, -0.5477343201637268, 0.1930241882801056, 1.5304043292999268, -2.3982839584350586, -0.0638008713722229, 0.9812202453613281, 0.042768869549036026, 0.4663950800895691, 1.061771273612976, 0.12933264672756195, -0.9167726039886475, -0.9947981238365173, -0.0288261529058218, -0.4228045344352722, 1.0723377466201782, 0.7421348094940186, 0.45405280590057373, 0.5006392598152161, -0.1788530945777893, 0.27748122811317444, 1.1823445558547974, -0.6841146349906921, 0.9673385620117188, -0.31412842869758606, -0.9035095572471619, 0.4826006293296814, -0.17059288918972015, 1.1252905130386353, -0.2920900583267212, -1.8387266397476196, 1.230591058731079, -0.09743911027908325, -0.1237807422876358, 1.7117587327957153, 2.039957284927368, -0.024440694600343704, -0.5706275701522827, -0.5373963713645935, 0.1962384730577469, 0.23197251558303833, 1.6581547260284424, 0.5889198184013367, -0.16911670565605164, 0.3703775703907013, -0.40511003136634827, 1.3535265922546387, -0.33504796028137207, 2.466148853302002, 0.5538064241409302, 1.379784345626831, 0.2967440187931061, -0.6595486998558044, -1.3078935146331787, -0.2697947323322296, -0.44855186343193054, 1.1542167663574219, -1.9513593912124634, -0.6290920376777649, 0.6246228814125061, 0.7764898538589478, -0.5616130232810974, -0.4614598751068115, -0.8215884566307068, -0.3471733033657074, -0.20931924879550934, 0.6786361336708069, 1.1029446125030518, -0.3338020145893097, 0.8058305382728577, 0.785582959651947, 0.6947665810585022, -1.7843793630599976, -0.9685110449790955, 1.0777989625930786, -2.310013771057129, -1.6643716096878052, -1.168219804763794, -1.6944619417190552, 0.9407410621643066, -0.15032640099525452, 0.36903634667396545, 0.7324714064598083, 0.2737824022769928, -0.6492401361465454, 0.06266345828771591, 0.020787304267287254, 0.14613981544971466, -1.0334986448287964, 1.449302315711975, -0.7146742343902588, 0.27938804030418396, -1.8799914121627808, 0.6173244118690491, -0.7975585460662842, -0.2541620433330536, -0.5438945889472961, -0.4918598532676697, -0.9981474280357361, 0.1551193743944168, -1.4764124155044556, -1.5435487031936646, 0.4668412208557129, -0.1554398238658905, -0.30492275953292847, -0.6514303088188171, 0.21414358913898468, 0.2529905140399933, -0.33012449741363525, -0.6357203722000122, 0.5495449304580688, -1.8076690435409546, 0.04329206794500351, -0.3650071322917938, 0.702537477016449, -0.3840043842792511, -2.4881770610809326, -0.9019636511802673, -0.5483415126800537, 1.296230435371399, -1.9521592855453491, -0.5796318054199219, 0.09897007048130035, -0.6061083078384399, 0.5731421113014221, 0.5432351231575012, 0.4594000279903412, 0.7654837369918823, 0.8724426627159119, -0.854259192943573, -0.2443724423646927, 1.2986668348312378, -0.9051439762115479, 0.8112840056419373, 0.5019584894180298, -0.9040820598602295, 1.9558857679367065, 0.07652293890714645, 1.3171439170837402, -1.0352659225463867, 2.267690896987915, -0.16926094889640808, 0.12960539758205414, -2.1974613666534424, -0.02632676251232624, -1.0584462881088257, -0.008305401541292667, -0.8708702325820923, 0.17487148940563202, -1.5046567916870117, 0.270081490278244, 1.0541963577270508, -0.2667887210845947, 1.8076050281524658, -1.6882283687591553, 0.6963043212890625, 0.7096303105354309, -2.060626983642578, 0.2908127009868622, -0.9904000759124756, 1.3742057085037231, -2.6113367080688477, 0.7536515593528748, -0.7403185367584229, 0.16554133594036102, 0.2605929374694824, 0.9760340452194214, 1.6257661581039429, 1.8507143259048462, 1.2119640111923218, -1.5161603689193726, 1.9185665845870972, -0.6933358311653137, -1.6891396045684814, 0.9119070172309875, -0.6187045574188232, -0.20394271612167358, 0.6450899839401245, -1.0778237581253052, 1.3801738023757935, 1.4167214632034302, -1.8147324323654175, 1.393936038017273, 0.33715444803237915, -0.13880454003810883, 0.3031378984451294, 1.1135473251342773, 0.5126930475234985, -0.7755993604660034, -0.28417477011680603, 0.8496271371841431, 0.6016765236854553, -1.2276592254638672, 0.44890493154525757, -1.778141736984253, 1.3387988805770874, 1.1525415182113647, 0.5955905914306641, 0.017591767013072968, -0.5946295857429504, 0.24571526050567627, -0.2931595742702484, 1.819594383239746, -0.36565446853637695, 1.4406368732452393, 2.3563175201416016, -0.6816152930259705, 1.1370667219161987, -1.8463926315307617, 0.11374588310718536, -1.144931435585022, 0.4319411516189575, -2.5297908782958984, -0.17077268660068512, 2.0521960258483887, -0.02906738966703415, 1.6981064081192017, 0.2607898712158203, 0.0424407422542572, -0.6922707557678223, 1.6974526643753052, 0.26198405027389526, -0.6153274178504944, 0.45519742369651794, 0.23046621680259705, 0.7033526301383972, 0.27967527508735657, -0.5240413546562195, -0.04611784964799881, -0.43181878328323364, -0.39148104190826416, 2.655235528945923, -0.8790608048439026, -0.1949469894170761, 1.1885027885437012, -1.8796648979187012, 0.09488330781459808, 1.467287540435791, 0.4341649115085602, -0.8434798121452332, 0.20775577425956726, 0.796497106552124, 1.3161568641662598, 0.3221237361431122, -0.024937400594353676, 1.120233416557312, 0.6836087107658386, 0.09611822664737701, 0.8364579677581787, 0.8508938550949097, -0.21045799553394318, -0.9681509137153625, -1.1081743240356445, -0.010902732610702515, -0.12323043495416641, -0.9295527338981628, 1.641595482826233, -0.22174596786499023, 0.3829059898853302, 0.2058006227016449, 0.13858076930046082, 0.612038791179657, 0.3816085159778595, 1.0788339376449585, 0.10757086426019669, -0.020897889509797096, -1.4497733116149902, -1.1585872173309326, 0.80972820520401, -0.634857714176178, 0.881059467792511, 0.863957405090332, -0.2879928946495056, -0.5434224605560303, -2.457148790359497, -0.6011140942573547, 1.0660932064056396, -0.6758855581283569, 1.6828913688659668, 0.37443193793296814, 0.6742373108863831, -1.5467400550842285, 0.23307594656944275, 2.5539052486419678, 0.5920778512954712, -0.3623791038990021, -0.69205641746521, -1.0138157606124878, 0.5934821963310242, 0.11009839177131653, -1.5721243619918823, -0.4735330641269684, 0.42849060893058777, -2.46443510055542, -0.21019428968429565, 1.8101463317871094, -0.7691488265991211, 0.4491898715496063, 1.0894359350204468, 0.5469104647636414, -0.028232671320438385, 0.47180551290512085, 1.2747117280960083, 0.07714814692735672, -0.5260125994682312, 2.039902925491333, -0.447028249502182, -0.6064936518669128, 0.18635490536689758, -0.7267022728919983, -0.11607583612203598, -0.024758651852607727, -2.0816831588745117, 0.44086953997612, -0.5990821123123169, -0.20604299008846283, -0.6011307835578918, 0.19332101941108704, -0.7036243081092834, -1.092584252357483, -0.5924941897392273, -0.2965487539768219, 0.19729577004909515, -1.7609169483184814, 0.22021275758743286, 0.05844393000006676, 0.8824036121368408, 0.3950306177139282, 0.6809408068656921, 0.697922945022583, -0.645508885383606, 0.4619656801223755, -0.5529152750968933, 0.3644963204860687, -1.3001525402069092, -0.9790872931480408, -0.5984790325164795, 0.4254623353481293, 2.4690468311309814, -0.8484895825386047, 1.1015973091125488, 2.3201348781585693, 1.207638144493103, -0.3018004298210144, -0.19074326753616333, 0.4167788624763489, 2.6091480255126953, 0.5030443072319031, -0.18561948835849762, 0.39673560857772827, -1.529835820198059, 1.2381913661956787, 0.984495997428894, -1.149737000465393, -0.34781646728515625, 0.8157044053077698, -0.8347703814506531, 0.3294760286808014, -0.011423865333199501, 1.134307622909546, 0.6220776438713074, 0.016852259635925293, -0.5249425172805786, 0.6263517141342163, -0.5934521555900574, 0.046221621334552765, -0.19008125364780426, 0.7779183387756348, -0.23817163705825806, -1.548766851425171, 0.0400652214884758, -1.0315241813659668, 0.1873885989189148, -0.8182286024093628, 0.9602147936820984, -1.055365800857544, 0.7332338094711304, 2.482051134109497, 0.3659929037094116, 0.4167463481426239, -1.0950695276260376, -0.5347806811332703, 0.9740148186683655, -0.417716383934021, 0.6523234248161316, -0.3378564119338989, 0.20723359286785126, -1.2809646129608154, -1.8970913887023926, -0.21604357659816742, -1.4916471242904663, 0.5312660336494446, 0.00744774891063571, -0.31618863344192505, -0.8534666299819946, 0.8623296618461609, -0.3201172947883606, -1.3135586977005005, -1.1597141027450562, -0.18617857992649078, 1.4276776313781738, 0.17153923213481903, 1.111732840538025, 0.38626691699028015, -0.5203160643577576, 0.23315474390983582, -0.31689026951789856, -1.2345795631408691, -0.39454662799835205, 0.07720600813627243, -1.7806586027145386, 0.20316468179225922, 0.3311135172843933, -0.29569730162620544, -0.44459420442581177, -1.2709763050079346, 0.6073858737945557, -2.030299186706543, -0.9313812851905823, 0.3238450586795807, 0.5261531472206116, -1.8001071214675903, 0.35620296001434326, 0.41877463459968567, 2.395216226577759, 1.7229431867599487, -0.6159180402755737, -1.6927434206008911, -0.615543007850647, 1.711678147315979, -0.5437480807304382, -0.47768256068229675, -0.4597768485546112, 1.398317575454712, -0.755165159702301, 1.03361177444458, 0.857803225517273, -0.02034478820860386, 0.8043917417526245, -0.8735703229904175, -0.0683756172657013, -0.006314537953585386, -0.642777144908905, -0.14411228895187378, 1.042715072631836, -0.22650699317455292, -1.5457698106765747, -0.5884430408477783, 0.9336395263671875, -0.34039443731307983, -0.44568997621536255, 1.8402308225631714, -0.3674887716770172, -1.6066030263900757, 0.05335090309381485, -0.1084349974989891, 0.23313915729522705, -0.5171292424201965, 0.5866262316703796, 1.6419435739517212, 1.0680429935455322, 2.5192794799804688, 0.5236425399780273, 0.2699710428714752, 0.5004869103431702, 0.7560548186302185, 1.0398166179656982, -0.29924044013023376, -0.7431039810180664, 0.7351574897766113, -1.4726332426071167, 0.3120543360710144, 0.8146021962165833, 0.05588110163807869, 0.38000017404556274, 2.2643496990203857, 0.3599563241004944, -1.5736380815505981, 0.4858873188495636, -1.0575954914093018, -0.610511839389801, 1.5472028255462646, -2.278554677963257, 0.4658195972442627, 0.3571925759315491, -0.34987887740135193, 1.131942629814148, -0.762360155582428, -0.32514190673828125, -1.4640955924987793, -0.5967457294464111, -0.09398739784955978, -0.17812204360961914, -1.702828049659729, 2.133298873901367, -0.7314590811729431, -0.21382325887680054, 1.433088779449463, -0.737460732460022, -0.4548071324825287, -0.12068463861942291, -0.11025501042604446, 1.3100626468658447, -0.40364307165145874, 0.012628510594367981, 0.7880039215087891, 0.6517283320426941, -2.274937629699707, 0.16229675710201263, -0.5226749777793884, -0.2763381004333496, -0.2707488238811493, 0.31939807534217834, 0.2201007753610611, 0.7200600504875183, 0.32886093854904175, -0.5323914289474487, 0.8005399703979492, 0.2763873338699341, -1.0995861291885376, 0.34997621178627014, -0.7903833389282227, -1.9908887147903442, 0.09540820121765137, -0.13822540640830994, -0.10731647163629532, 0.13954833149909973, 0.6324572563171387, -0.472819983959198, -0.1358850747346878, 1.5593539476394653, -1.7859536409378052, 0.9215267896652222, 0.5509392619132996, 0.5098304748535156, 0.7896966934204102, -1.433320164680481, 1.119063138961792, 0.8690212368965149, 1.2125643491744995, -0.004130208864808083, -1.7527713775634766, -0.4726753234863281, 1.608200192451477, 0.6402126550674438, 0.881163477897644, -0.023169316351413727, 0.835583508014679, 0.7873338460922241, -0.7989153265953064, 0.31381550431251526, 1.2124208211898804, 0.662084698677063, -0.06394246220588684, -0.27481260895729065, -0.22342926263809204, -1.09061598777771, -0.4283415675163269, 0.4346994459629059, 0.2974220812320709, -0.7465357780456543, -0.32704970240592957, -1.7407259941101074, -0.5751101970672607, -0.7417398691177368, 1.400499939918518, -0.1584574282169342, -1.2149070501327515, -0.10212163627147675, -1.1228644847869873, 0.4742760956287384, -0.886099100112915, 1.0851364135742188, -1.1507737636566162, 1.9468741416931152, 1.2411444187164307, -0.11557299643754959, 0.5343263745307922, 0.8346025943756104, -1.20053231716156, -1.3785866498947144, -0.6374272704124451, 0.18084530532360077, 1.027613878250122, -0.5442208647727966, -1.6664072275161743, 0.4529998302459717, -0.6137967109680176, -0.4541817307472229, -0.6126608848571777, 0.7352690696716309, 0.9168618321418762, -0.6369809508323669, -0.28530946373939514, 1.0307772159576416, 0.6401082277297974, -0.7421078681945801, 0.2002338320016861, -0.6691608428955078, -0.8789385557174683, -0.7300122976303101, 0.6902035474777222, -0.552243709564209, -0.16470368206501007, -0.3303124010562897, 1.6984257698059082, 0.4491164982318878, 1.0225744247436523, 0.7384091019630432, -1.7377506494522095, -1.700058102607727, -0.38321614265441895, 1.0703134536743164, -0.8290451765060425, 1.9558541774749756, -0.8420912623405457, 0.38246387243270874, 0.9956575632095337, 0.6493865251541138, 2.615841865539551, -1.0285823345184326, -0.7844623327255249, -1.067112922668457, -0.09634006023406982, 0.2865891456604004, -1.1248033046722412, -0.14635805785655975, -1.2050056457519531, -0.19402015209197998, -1.2918362617492676, -0.22630831599235535, 0.7201986312866211, 2.6726913452148438, 1.3509694337844849, 1.9035829305648804, -0.5586947202682495, 0.3069392740726471, -1.3504618406295776, 0.11784810572862625, 0.18720166385173798, -0.04767559468746185, -0.25654906034469604, -0.24503779411315918, -1.405267596244812, 0.8321681618690491, -1.5247395038604736, 1.2725142240524292, 2.362755537033081, 0.6732308268547058, 1.2292256355285645, 1.0927659273147583, 0.11564094573259354, -0.8056877255439758, 1.380615472793579, 0.617252767086029, -1.3487951755523682, 2.324336051940918, 0.000554643920622766, -0.1904648393392563, 1.4045320749282837, -1.098082423210144, 0.33878329396247864, -0.2687760293483734, 1.0354435443878174, 0.9601209759712219, -0.027518849819898605, 0.6372043490409851, -0.7531699538230896, 1.614313006401062, -0.4523398280143738, 2.219609022140503, 1.4098296165466309, -0.5123049020767212, 0.1341501772403717, -0.43220147490501404, -0.5005386471748352, -1.3125629425048828, 1.122334599494934, 0.034279756247997284, 1.2602163553237915, -1.359312891960144, 0.35173487663269043, 0.33115074038505554, -0.1753433793783188, -0.3340131640434265, -1.3062254190444946, -1.1350816488265991, 0.06717603653669357, 0.7249214053153992, -1.0771478414535522, 0.9502790570259094, 1.386593222618103, 1.81789231300354, 0.046884309500455856, 0.38175541162490845, -0.16924773156642914, 0.5110825896263123, 0.3418426513671875, 0.9570728540420532, -0.04596177488565445, 2.350167751312256, 2.737898111343384, 0.48401370644569397, 0.4158955514431, -2.738220453262329, -0.7009825706481934, 0.3285938501358032, -0.4692489504814148, -1.2939993143081665, -0.6053870320320129, -1.533073902130127, -0.8615180850028992, -1.3571964502334595, 0.8099364638328552, -0.6655254364013672, 0.49598050117492676, -0.8587110638618469, -2.6734678745269775, 1.086861491203308, -0.42369544506073, -1.760797142982483, 0.5929046869277954, 0.2831612229347229, 1.6100436449050903, -2.0437567234039307, 0.6539827585220337, 0.670005202293396, 1.7168501615524292, 0.18879461288452148, 0.0064153545536100864, 0.44816648960113525, 0.9971010684967041, -0.7777823805809021, -0.36404451727867126, 0.06783950328826904, 2.117497205734253, 0.7238684892654419, -0.4259066879749298, -0.35492265224456787, 0.7403843402862549, -0.21264098584651947, 0.016337387263774872, 1.2660290002822876, 0.19151397049427032, -1.1686997413635254, -0.23258277773857117, -0.725020170211792, -2.094700574874878, 0.8474724292755127, -0.17874644696712494, -1.2274571657180786, -0.256634384393692, 1.3919185400009155, 0.6487326622009277, -1.6928977966308594, -1.1833115816116333, -0.454328328371048, -0.6024535298347473, -0.7045175433158875, 0.34706342220306396, -1.3690708875656128, 0.5262356996536255, -0.31826162338256836, -2.6478848457336426, -0.7370880842208862, 1.1392455101013184, -0.6210356950759888, 0.4206545650959015, -0.7444294691085815, -0.6422469615936279, 1.3506202697753906, -0.40490004420280457, 1.406951665878296, -1.3647751808166504, -1.2835644483566284, 0.7502433657646179, -0.617031991481781, -1.907433032989502, -0.31317490339279175, 0.5598755478858948, -1.0828100442886353, -0.45081618428230286, -0.7869410514831543, 0.10274891555309296, 0.8788118958473206, 1.3336416482925415, 0.47675055265426636, 1.7503612041473389, 1.0339819192886353, 0.8518677949905396, 2.8453850746154785, 0.5112926959991455, -2.0193591117858887, 0.2248259037733078, 0.5239627361297607, 1.573713779449463, -0.2592567503452301, -1.3552830219268799, 1.5257118940353394, -1.782239317893982, -0.6819952130317688, 0.0019204550189897418, -0.2772400677204132, -0.620817244052887, -0.11146479099988937, -0.2253289818763733, -0.03359154239296913, -0.06265904009342194, 0.3569878041744232, -1.5373131036758423, 0.6355558037757874, -0.49746841192245483, 1.6196774244308472, 0.5195387601852417, 0.44457361102104187, 0.9652038812637329, 1.5232782363891602, -0.8497411012649536, 2.0150487422943115, -0.9142451882362366, -0.18529248237609863, -2.6060678958892822, -0.13325268030166626, -0.6261572241783142, -1.2521133422851562, 0.5398993492126465, -0.12806542217731476, 0.8739933371543884, 0.041498251259326935, -0.33206647634506226, -1.7996933460235596, -0.5237593054771423, 1.6870906352996826, 0.055493712425231934, 0.5010798573493958, -0.7944582104682922, -0.8961222171783447, -2.7573320865631104, -0.1486087292432785, 1.0591179132461548, -0.13859926164150238, 0.46777835488319397, 1.5711392164230347, 0.8156782388687134, 0.555195152759552, -0.4282125234603882, -1.3700449466705322, 0.8343355059623718, 0.5457434058189392, 0.4079195559024811, 1.2699179649353027, 1.4197689294815063, -0.6048231720924377, 1.8958579301834106, 0.576977014541626, 1.5804401636123657, 0.4697321057319641, -1.3207521438598633, -0.4347495436668396, -0.9245378971099854, -0.9975572228431702, -0.4105556011199951, -0.20642691850662231, -1.309094786643982, 0.8024916052818298, -1.2564901113510132, 0.11261216551065445, 0.7793658375740051, -0.9835324287414551, 0.18078911304473877, -0.2270229607820511, 0.36190855503082275, -0.6530680060386658, -0.4994262456893921, -0.09740859270095825, -2.160350799560547, 1.4524136781692505, -0.05049913749098778, -0.5187966227531433, -0.38949957489967346, -0.06154419481754303, -0.40898069739341736, 0.902941107749939, 0.7560032606124878, -1.0048749446868896, 2.329063892364502, -0.3323015868663788, 0.3717546761035919, 0.3102814853191376, -0.7860833406448364, -0.8849588632583618, 0.4768982231616974, -0.5512651205062866, -0.9747990965843201, 0.39652419090270996, -0.36769020557403564, -0.8544052839279175, -1.3948378562927246, 1.4398328065872192, 0.14852531254291534, 0.21938525140285492, -0.7151271104812622, -1.1817883253097534, -0.6932358741760254, -0.9888993501663208, 1.5197505950927734, 1.2485086917877197, -1.0119452476501465, 0.31098267436027527, 0.0730556771159172, -0.2980312705039978, 0.6055465340614319, 1.76613187789917, -1.40872061252594, 1.838003396987915, -0.9823170304298401, 0.8104283809661865, -2.094634771347046, -1.3053334951400757, -0.7838232517242432, -0.38537976145744324, -1.0398175716400146, 1.378096580505371, -0.04903130978345871, -0.06510413438081741, -0.7050612568855286, -0.5080229043960571, 1.7176291942596436, 0.5405524373054504, -1.333391785621643, -0.5393602848052979, -0.06542752683162689, 0.04346923157572746, -2.546015501022339, 0.5742694139480591, -0.13719478249549866, 0.45576542615890503, 0.9157270193099976, -2.066875696182251, -0.1044798195362091, 0.2860342264175415, 0.7463200688362122, -0.5004851818084717, 1.3462074995040894, -0.8951261639595032, -0.7850489616394043, 0.30463215708732605, -0.19690148532390594, 0.5695938467979431, -0.36440443992614746, 0.9236817955970764, -0.6301764845848083, 2.0709054470062256, 0.07667003571987152, 0.5045773386955261, 0.9822272658348083, 1.022661805152893, -0.9202176332473755, 1.1712515354156494, -1.3138259649276733, -1.7125109434127808, 0.2851261794567108, 1.2971458435058594, 0.5767065286636353, 0.4784191846847534, 2.317802906036377, 0.9487671256065369, 1.0537559986114502, 0.005116203334182501, 0.7850633263587952, -0.8135364055633545, -0.18507196009159088, -0.6550030708312988, 2.1699349880218506, -1.3611587285995483, -0.3543636202812195, -0.6521251201629639], [-0.06466832011938095, 0.9068276286125183, -0.6211201548576355, -0.5526735186576843, -1.4136825799942017, -0.3538666367530823, -0.8795806765556335, -0.7257122993469238, 1.2770662307739258, 1.0951805114746094, -0.8577565550804138, 0.09927776455879211, -1.6538516283035278, 0.5702559351921082, -0.6415780782699585, 0.010732701979577541, -0.2034219205379486, 1.547838568687439, 0.1790328323841095, 1.0852010250091553, 0.08567752689123154, 1.2591158151626587, -0.07421105355024338, -0.4871900677680969, 0.40778273344039917, -0.88979572057724, -0.8736629486083984, 0.4366409182548523, -1.1626530885696411, 0.1943846195936203, 0.3565179407596588, -2.269866704940796, -0.06538043916225433, 1.3643587827682495, -1.0396422147750854, -0.15763431787490845, 1.9798290729522705, -0.6099448800086975, 0.4898035526275635, -0.9004460573196411, 1.3737276792526245, -1.507333517074585, -2.3110337257385254, 0.7144652009010315, -0.7156583666801453, -1.9734611511230469, 1.2628101110458374, 0.3753818869590759, -0.7171537280082703, -0.5398156642913818, 0.7087605595588684, -1.1152633428573608, -0.7134265899658203, 0.36942997574806213, -0.5146492719650269, 0.13942056894302368, -1.1884034872055054, -0.30309244990348816, 0.4367654323577881, 0.5012702941894531, 0.08757197856903076, 1.3345454931259155, 0.20950724184513092, -1.109099268913269, 1.0587319135665894, 0.913043737411499, -0.24675992131233215, -0.5534018874168396, 0.3905341327190399, 0.7811343669891357, -2.261425256729126, -0.2454589456319809, -1.1238797903060913, -1.4587477445602417, 0.4425402283668518, 0.5189706087112427, -0.08598752319812775, -0.11198564618825912, 1.7654024362564087, 0.562362790107727, 0.34045711159706116, -0.4927116334438324, -0.48543137311935425, -0.4693668484687805, -0.007160968147218227, 0.823204755783081, -0.19578541815280914, -0.3858661651611328, -0.7448340058326721, -1.7988280057907104, 0.5377058982849121, -0.7257087826728821, -1.9890888929367065, -0.3300440013408661, 2.210684299468994, -0.3033141493797302, 0.9998465180397034, 0.08322575688362122, -0.5198033452033997, 0.6011877059936523, -0.30805671215057373, 0.5833699107170105, 0.13053661584854126, -0.8464334011077881, -0.26855841279029846, -0.5976369380950928, 1.0661762952804565, -1.8700677156448364, -0.6865056157112122, 0.21730080246925354, -0.9987262487411499, -0.8909287452697754, -1.2498379945755005, 0.1998843401670456, 0.009749406948685646, -2.394606828689575, 0.0589853972196579, -1.1421951055526733, -0.5480288863182068, -0.34296923875808716, -0.683536171913147, -1.1072717905044556, -1.2668991088867188, 0.2517825961112976, 1.216689944267273, 0.5403924584388733, -0.3126337230205536, 0.1511557251214981, -1.6390044689178467, -0.5398153066635132, 0.3463344871997833, -0.9326059818267822, 2.0338141918182373, 0.2602126896381378, 0.36207714676856995, -1.3754407167434692, 0.8600482940673828, 1.0181498527526855, -0.9678519368171692, 0.8374251127243042, -1.4996298551559448, 1.3634082078933716, -0.525157630443573, 0.4473319947719574, 2.52591609954834, -0.29999688267707825, 1.3906034231185913, 0.8456960320472717, 0.814446210861206, -0.8427406549453735, 0.825412929058075, -1.4273799657821655, 1.045554757118225, 0.9536499381065369, 2.3099701404571533, 0.5112358927726746, -0.5174894332885742, 0.9862247705459595, -1.2910209894180298, -0.4843882918357849, 1.7085480690002441, 0.5102877020835876, -0.3823294937610626, -1.4447369575500488, 0.42328256368637085, 0.4610641896724701, 0.5955451130867004, 0.2343306839466095, 1.347082257270813, 1.3673062324523926, -0.3072827458381653, -0.4347775876522064, -0.9950180053710938, -0.11155004799365997, 0.13704735040664673, 0.07493313401937485, 2.0137884616851807, 0.8681119084358215, 0.2460223287343979, -0.9467719197273254, -1.227601170539856, -0.9617941379547119, -0.9571361541748047, -0.06755703687667847, 0.9268807172775269, 0.2499825358390808, 0.043995268642902374, -0.7337372303009033, -0.08412649482488632, 0.4534079134464264, 0.46042293310165405, 2.1441562175750732, -0.06732422858476639, 0.8665235042572021, -0.6170397996902466, -0.28359565138816833, 0.63071608543396, 1.1145590543746948, -0.18576666712760925, 0.400269091129303, 0.7283473014831543, 0.24817396700382233, -0.7017257213592529, 0.17379122972488403, -0.8515515923500061, -0.5692191123962402, 0.009994769468903542, 0.43603479862213135, -0.30303868651390076, -2.6148741245269775, -1.090035319328308, -1.323367714881897, 0.13188637793064117, -0.039780471473932266, 0.54819655418396, -0.9868422746658325, 1.1220757961273193, 0.963252067565918, -1.0726985931396484, 0.7792366743087769, -0.6101652979850769, 0.833791196346283, -1.0800589323043823, 0.6993947625160217, -1.0006134510040283, 1.2586524486541748, 0.22174647450447083, -0.5039567947387695, -0.18199309706687927, -0.5167286396026611, 0.2885668873786926, -0.14699283242225647, -0.37031838297843933, -0.3371317684650421, -1.7119418382644653, -1.8975512981414795, -0.020151788368821144, -1.182493805885315, -1.4046854972839355, 0.17860035598278046, 1.0698096752166748, 1.4185324907302856, 0.26187992095947266, -1.11525559425354, 1.4006894826889038, -1.245524525642395, -1.1127033233642578, 0.22413505613803864, 0.17145052552223206, -0.2190953493118286, 0.35925203561782837, 0.7238406538963318, -0.7000913023948669, -0.636931300163269, 2.334827184677124, 1.0994466543197632, -0.03471030667424202, -1.9754416942596436, -1.014488935470581, -0.24616873264312744, -0.03678152337670326, 0.9454582929611206, -0.3814416825771332, -0.010846047662198544, 0.24020454287528992, -0.8294911980628967, 0.02230810932815075, -1.01376473903656, 0.5343178510665894, -1.4363125562667847, -1.6569247245788574, -1.4673818349838257, 0.029603397473692894, 0.1438029557466507, 0.8892725706100464, -0.37748634815216064, 1.5671577453613281, 0.2523036599159241, -0.7280958294868469, -1.120247483253479, 1.691245675086975, 1.8574339151382446, -0.26261284947395325, -1.1740666627883911, 0.6386407613754272, -0.8342638611793518, -1.1616365909576416, -2.1262736320495605, -0.18617860972881317, -1.2646957635879517, 0.11837365478277206, -1.587907314300537, 0.6607232093811035, -0.9622294306755066, 0.9953501224517822, 1.0299962759017944, 0.606618344783783, 0.5859403610229492, -0.5312068462371826, -0.2539129853248596, -2.401705026626587, -0.39631763100624084, 0.11613570898771286, 2.1292028427124023, -0.0744626596570015, 0.05306728929281235, 0.7943416833877563, -1.0458191633224487, -0.30394384264945984, -0.27862489223480225, -2.586538314819336, -0.6874098777770996, -0.3469957411289215, 1.1368727684020996, 0.23563173413276672, 1.0990897417068481, 0.724612832069397, 0.8439458608627319, -2.161482572555542, 2.8975749015808105, -0.2243366688489914, 0.0641518384218216, 0.940102756023407, -0.9208557605743408, 0.22011739015579224, 0.49060162901878357, -0.6523006558418274, -0.17750205099582672, -0.7045845985412598, -0.45645448565483093, 1.215125560760498, -1.7712578773498535, 0.7276584506034851, -0.8865602016448975, 1.811667561531067, 1.081117868423462, 0.5757085680961609, -0.3285900056362152, -0.9152043461799622, -2.513441562652588, -0.4780607521533966, -0.5220654010772705, -0.09622872620820999, 0.10418921709060669, 0.5321463346481323, -0.08921229839324951, 0.1438222974538803, -0.8441764712333679, -1.9716627597808838, -1.4499647617340088, -1.4245786666870117, 0.8601906895637512, -1.1956592798233032, 1.5242815017700195, -0.25739604234695435, -0.629553496837616, 0.21445946395397186, -0.9856942892074585, -0.6242304444313049, -0.48246708512306213, -0.2798503041267395, 0.670372724533081, 1.3593937158584595, 0.16151908040046692, 1.4239609241485596, -0.07992255687713623, 0.9485704302787781, 0.4316287040710449, -1.0945297479629517, 0.6865697503089905, 0.2800946831703186, -0.5871860384941101, 0.14779897034168243, -0.4861564636230469, -0.09027723968029022, -0.15188093483448029, 0.2709364593029022, -0.33875152468681335, 0.2969200611114502, -1.7481766939163208, 0.4715118706226349, 1.0006077289581299, -0.7551301121711731, 0.5547172427177429, -1.0342814922332764, -0.864871621131897, 0.4060564637184143, 0.5486952066421509, -0.33980637788772583, 0.29775193333625793, 0.15796653926372528, 0.7160606980323792, 0.4928975999355316, 0.7037985920906067, 1.3954371213912964, 0.9549856781959534, 0.5652148723602295, -1.6103339195251465, 0.24278397858142853, -1.272149682044983, 0.5721307396888733, -1.6523886919021606, 1.2961819171905518, -1.061963677406311, 1.3922390937805176, -0.44643470644950867, 0.15368211269378662, 1.3261809349060059, 0.23604142665863037, 0.4040074348449707, -0.03240926191210747, -1.1946865320205688, -1.8866596221923828, -0.7872483730316162, 0.7917897701263428, 0.5617026686668396, 2.3840155601501465, 1.547549843788147, 1.1453965902328491, -1.6171183586120605, 1.2918044328689575, -1.7993555068969727, -2.4062390327453613, 1.6628001928329468, -0.4781632125377655, 1.3931150436401367, -0.8000181913375854, 1.6596349477767944, 0.11408818513154984, 0.6954344511032104, -1.815066933631897, -0.5258160829544067, 0.369632363319397, 0.17006875574588776, -0.5999494791030884, 1.329899549484253, -0.8082159161567688, 0.5130009651184082, 0.49851125478744507, 0.4838877320289612, 0.6788895726203918, 1.301837682723999, 0.07728935778141022, -1.0218627452850342, 0.0504213385283947, 0.6463699340820312, 0.9116731286048889, 0.3166850805282593, 0.9201928377151489, -0.32317495346069336, -0.5655134916305542, 0.3377169072628021, -1.0094974040985107, 0.6539003849029541, -1.660977840423584, 0.11803409457206726, -0.8367959260940552, -1.0362532138824463, -2.0977249145507812, -0.5573630332946777, -1.4368938207626343, -0.29232317209243774, 0.47464844584465027, -0.25221917033195496, 1.1232757568359375, 1.2939891815185547, 0.2904106676578522, 1.9836907386779785, -0.02676754631102085, 0.21285493671894073, -1.0652575492858887, -0.7239599227905273, 0.4291706085205078, -0.5838756561279297, -0.8581357002258301, 0.5306380987167358, -0.5600325465202332, -0.7938368916511536, -0.9662225842475891, -0.7912405133247375, 0.5818767547607422, 0.6284714341163635, 0.6579146981239319, 0.31131139397621155, 0.36312592029571533, 0.1039542630314827, -0.43125230073928833, 2.0808358192443848, 0.09177841246128082, 0.7002096176147461, -1.1711522340774536, -0.38566479086875916, 0.787742555141449, 1.6105550527572632, 0.7090404033660889, 0.6284288167953491, 0.7900034785270691, -0.938886284828186, 0.14924876391887665, 0.7357559204101562, 0.3210049569606781, -0.24384498596191406, 0.20147784054279327, 2.3925375938415527, -1.1497039794921875, 0.8693972229957581, 1.081695795059204, -0.19842389225959778, 0.5871005654335022, 1.1783208847045898, 1.0915625095367432, 1.5162018537521362, -1.8073651790618896, 0.47249549627304077, -0.45373907685279846, -1.0162259340286255, 1.1414917707443237, -0.8073158860206604, 0.003063530893996358, 1.6614227294921875, -1.177430272102356, 0.2319881170988083, 2.9565412998199463, 0.3630301058292389, -0.027895933017134666, 1.7375802993774414, 1.3273142576217651, -0.2666015028953552, -1.4490330219268799, -0.3044370114803314, 0.10857265442609787, -0.8990966081619263, -0.0586363710463047, -0.7326736450195312, 0.668159008026123, -0.8700282573699951, -0.5306023955345154, 0.09484705328941345, -1.4400086402893066, -1.0778844356536865, -1.640162706375122, -0.49185994267463684, 0.8021995425224304, -0.2850255072116852, 0.32713204622268677, 0.631068766117096, 1.9772495031356812, 0.514654278755188, 1.129273772239685, -0.6193963885307312, 1.1189769506454468, 0.9919177293777466, 0.6639620065689087, -0.9722960591316223, -0.6241999268531799, 1.857911467552185, -0.6410576105117798, -0.04187933728098869, 0.2724798917770386, 1.6447782516479492, 0.3221776485443115, 0.36425215005874634, 1.2768361568450928, -0.8768047094345093, 0.19665022194385529, -0.08956404030323029, 0.5861205458641052, 1.638769507408142, 2.0707550048828125, -1.3046070337295532, 0.756522536277771, -0.07637326419353485, -0.8070430159568787, -0.7331900000572205, -2.091550588607788, 1.5126992464065552, -0.8641864657402039, -1.2556066513061523, 0.589673638343811, 1.0882904529571533, -0.2569076716899872, -0.7564131021499634, -0.24953874945640564, -2.1904513835906982, 0.7235668301582336, -0.6401849389076233, 0.4040524959564209, 0.7772784233093262, 0.35820984840393066, 0.31677865982055664, -0.04055717960000038, -2.326833486557007, 0.883877694606781, 1.7769153118133545, 0.16477912664413452, -0.7027308940887451, 0.2380320280790329, -0.9683363437652588, 1.534222960472107, 1.1970800161361694, 1.1637496948242188, 0.4941954016685486, 1.454162836074829, -0.17418219149112701, 1.3870519399642944, -0.6272020936012268, -0.6290621757507324, 0.07556246966123581, 0.7662511467933655, -0.35344988107681274, 1.1861203908920288, -0.21288691461086273, -0.8123521208763123, 2.1934096813201904, -0.7454175353050232, 0.09122234582901001, -1.0133366584777832, 1.1381434202194214, 0.28775477409362793, 0.7044970393180847, -0.7944201827049255, -1.376145362854004, -0.5321283936500549, -0.14831751585006714, -0.8067945837974548, 0.4508642852306366, -0.9772899746894836, -0.6139254570007324, -0.2873137593269348, 1.159109354019165, 0.17686817049980164, 0.646921694278717, 0.1604873538017273, 0.03271022439002991, -0.4148626923561096, -0.8605266809463501, 0.7983618378639221, -0.7593463659286499, -1.1226136684417725, 0.4773622453212738, -1.0299700498580933, 0.5356787443161011, 0.1979796588420868, 0.43143460154533386, 0.7895826101303101, -0.42746901512145996, 0.4713813066482544, 0.11204708367586136, 0.6863459944725037, -0.840530514717102, 0.517317533493042, 1.3616869449615479, -0.4525701105594635, 0.01194281131029129, -0.9260299205780029, -0.7288920283317566, -0.7149522304534912, -0.21024039387702942, -1.3008294105529785, 1.4430729150772095, -0.02847699262201786, 1.4790939092636108, -0.44748106598854065, -1.1500000953674316, -1.2085152864456177, -0.834168016910553, 0.5890510082244873, 0.6688673496246338, -1.0015884637832642, 0.5639277100563049, -0.9731135964393616, 0.8154308795928955, 1.0899330377578735, 1.2255445718765259, -1.1639113426208496, 0.3534677028656006, 0.7050854563713074, 1.1528866291046143, -2.028585195541382, -0.06975343078374863, -0.2276076376438141, -0.6250685453414917, 1.1076672077178955, -0.7563045024871826, 1.033216118812561, -0.8540173172950745, -1.4874496459960938, 0.7025324702262878, 0.6076029539108276, -0.3951804041862488, -0.6061856746673584, -0.9306719899177551, 1.106475830078125, -1.9312304258346558, -1.8334084749221802, -0.7972097992897034, -0.5818329453468323, 1.0969330072402954, -0.19674666225910187, -0.19868674874305725, -1.2854920625686646, 0.2503645122051239, -0.571772575378418, -0.005985697265714407, 0.7387571930885315, -0.264995276927948, -0.6287553310394287, 0.4081858992576599, -0.6669658422470093, 1.2992033958435059, 0.40522515773773193, -0.010230551473796368, -0.34817683696746826, 0.8623554110527039, -0.2660105526447296, -0.24665389955043793, -0.21593700349330902, -0.6789126992225647, -1.7258381843566895, -0.399944543838501, -1.9748600721359253, 0.6306912899017334, 2.636213779449463, -0.13462205231189728, -0.050224900245666504, -0.5962789058685303, 1.7368398904800415, -0.8780422210693359, 0.14991708099842072, -0.40392324328422546, 0.29857921600341797, 0.45856520533561707, 0.8218952417373657, -1.1235804557800293, 0.013582736253738403, -0.7655397057533264, 0.9571364521980286, -0.6512904167175293, 0.3926345407962799, -0.4395509362220764, 0.9419741034507751, -0.71780925989151, -0.06993331760168076, -1.6704177856445312, -0.47213974595069885, 1.2516016960144043, -1.1596580743789673, 1.3402737379074097, -0.29388105869293213, -0.7780249714851379, -0.2257656306028366, -0.5887568593025208, 0.7980734705924988, 1.202649474143982, -0.23279808461666107, -0.8131706714630127, -1.1930160522460938, 0.3198343813419342, -0.8254080414772034, 0.5850389003753662, 0.3442929685115814, 0.48184701800346375, -0.263087660074234, 0.5765306949615479, 0.9732813835144043, 1.0996654033660889, -0.001064867596141994, -0.22702662646770477, -0.5545253753662109, 0.5815243721008301, 0.5932700037956238, -0.5575695037841797, 1.9899189472198486, 0.30655160546302795, 1.0702824592590332, -1.122562289237976, -0.4731477200984955, 1.1112887859344482, -0.7174780368804932, -0.7345671057701111, -0.4896087944507599, 0.21467171609401703, -0.4741460680961609, -0.1294129490852356, 1.4882320165634155, -0.7412928938865662, 0.3325912058353424, 0.3666914105415344, -1.1705071926116943, -0.8637365102767944, 0.7796761393547058, 0.9173932075500488, 1.4692013263702393, 0.04947594180703163, -0.24535997211933136, -0.7469164133071899, -0.07007437944412231, 0.33987775444984436, 0.6321102976799011, 0.05505220964550972, -0.22180360555648804, 1.135669231414795, -0.4743855595588684, 0.09637583792209625, 0.6963815689086914, -0.9778614044189453, 0.5520935654640198, -1.972019910812378, 0.09594257175922394, -0.9382613301277161, -0.6200266480445862, -0.16463373601436615, 0.869183361530304, 2.0002338886260986, -0.4478427469730377, -1.1644296646118164, 1.3908005952835083, 0.39869603514671326, -0.2968154847621918, 0.17695552110671997, 0.25051385164260864, 0.4105900824069977, -0.6599572896957397, 1.525053858757019, 1.2933424711227417, 1.6429134607315063, -0.2835518717765808, -0.4325403869152069, 0.9762667417526245, 2.9368209838867188, 0.3344385623931885, 0.589575469493866, 1.1176517009735107, -0.14007477462291718, 1.0405569076538086, -0.07176440209150314, 0.7563583850860596, -1.889809489250183, 0.9005219340324402, -0.252357542514801, -0.9707005023956299, 1.3057609796524048, -1.4777858257293701, 0.3947754502296448, 0.7567574381828308, -0.1165677160024643, 3.3359122276306152, -0.25352421402931213, 0.9577754735946655, -0.12230809777975082, -0.5482451915740967, 0.6035497188568115, 0.12798942625522614, -0.8545315861701965, 0.8733104467391968, -0.648905336856842, -0.9394046664237976, -0.5051303505897522, 1.1721241474151611, -1.0976117849349976, -0.04998745024204254, 0.09441570192575455, -2.355217218399048, -0.08950338512659073, -0.2974180579185486, -1.3104099035263062, -1.015271782875061, 0.2403961718082428, -0.48831337690353394, -1.3199021816253662, 1.2713865041732788, 0.9412089586257935, -0.42049074172973633, 0.9815092086791992, 0.11811138689517975, 0.7749698758125305, -2.059340000152588, -0.8853879570960999, -0.05520199239253998, 1.7871173620224, 0.09239353239536285, 0.5869845151901245, -1.0217434167861938, -0.7498927116394043, 1.0869536399841309, 0.8514622449874878, 0.42660656571388245, 0.0260769035667181, -0.6851642727851868, 0.1779811680316925, -0.5488243103027344, -1.3196207284927368, 1.137178897857666, 0.15616129338741302, 1.8070640563964844, 0.4656108021736145, 0.08534978330135345, 1.372076392173767, -0.6034700274467468, 0.07452315837144852, -0.04869914427399635, -0.5097531080245972, -0.375762939453125, 1.001980185508728, 0.30903980135917664, 1.025916337966919, -0.5923978686332703, 0.47232720255851746, 0.581009030342102, -0.33516719937324524, 1.9460159540176392, 0.0277631226927042, -0.8903993368148804, -1.0782432556152344, -1.7739616632461548, -0.29242125153541565, -0.5467589497566223, 1.3624403476715088, 1.6721291542053223, -0.04678041860461235, 0.42346125841140747, 0.6122249960899353, 0.7274700403213501, -0.19952484965324402, 0.9380621910095215, -1.464259147644043, -2.1890339851379395, 0.49172651767730713, -0.23211780190467834, 1.5259212255477905, 0.059361159801483154, 0.7948294878005981, 0.33202245831489563, 0.6913063526153564, 0.43894556164741516, -0.2412337213754654, -0.19670632481575012, 1.0223091840744019, 0.6617524027824402, -1.528047800064087, -1.8809691667556763, -1.2485158443450928, -0.22780416905879974, 2.5010130405426025, -0.7949775457382202, 0.8527270555496216, -1.796248197555542, -0.3299277722835541, 1.9093612432479858, -0.03431663289666176, 0.8066914081573486, -1.4109883308410645, 0.8389026522636414, 0.4991772770881653, -1.2963546514511108, 0.11554944515228271, -0.4387890696525574, 1.48836350440979, -0.42562422156333923, 1.1287362575531006, 0.9493114948272705, -1.550324559211731, 0.6820719242095947, -0.10761842131614685, 0.08317171782255173, -0.11441154778003693, -1.3336966037750244, -0.7581473588943481, 0.7300257086753845, 1.7986438274383545, 0.8764374852180481, 0.10540790855884552, 0.4499582052230835, -0.510030210018158, 0.5443640351295471, -0.8706899285316467, -1.334942102432251, 1.3832788467407227, -0.4612278938293457, -1.2530666589736938, -0.4919690787792206, 0.1121257096529007, 0.13658693432807922, 1.0999317169189453, 0.7704711556434631, -0.4190465807914734, -1.2588268518447876, 0.6120297312736511, -1.8763525485992432, 0.5697629451751709, -0.12279805541038513, -0.4314115345478058, -1.50277841091156, -0.9041575193405151, 1.472731351852417, -0.46906766295433044, 0.24036310613155365, 0.22618940472602844, -0.6528837084770203, -0.2927573025226593, 1.2181977033615112, -1.6041516065597534, 0.7409126162528992, 1.1986418962478638, -1.66782546043396, -2.2582573890686035, 0.15915751457214355, -0.017463602125644684, 0.45477375388145447, 0.14213275909423828, -1.5808982849121094, 0.3359006345272064, 0.608833372592926, -0.31386590003967285, -0.001749006798490882, 0.5104507207870483, 0.270998477935791, 1.707841396331787, -0.364810973405838, 1.173654556274414, -1.5278044939041138, 1.8259953260421753, 1.7290796041488647, -0.7613943815231323, -1.0578819513320923, -0.1617116630077362, 0.014190602116286755, 0.05841239169239998, -1.4833110570907593, 0.15109778940677643, -0.5739361047744751, -1.8256162405014038, -0.3970910608768463, 1.5042012929916382, 0.5816569328308105, -1.1885117292404175, -1.2224252223968506, -0.9960776567459106, -0.5229406952857971, -0.29801130294799805, -0.5899657011032104, -0.8695124983787537, 0.3713545501232147, 0.08500278741121292, 0.589502215385437, 1.1999222040176392, 0.02002224512398243, 0.358510285615921, -0.5097205638885498, -0.3230575621128082, 0.3518945872783661, -0.14336830377578735, 1.1286166906356812, -0.3409464657306671, 0.10877935588359833, -0.7421388626098633, -0.34990301728248596, -1.8033159971237183, -0.7757771015167236, -1.279096007347107, -0.23128172755241394, -0.9147210121154785, 0.13685114681720734, -0.01645077019929886, 0.01095338724553585, -1.1072779893875122, -1.19899582862854, 1.2983072996139526, -0.26344141364097595, 0.8645108342170715, -0.362091988325119, 2.5309276580810547, -0.052415262907743454, 0.6508387923240662, -1.1858325004577637, -0.2680654525756836, 0.8889367580413818, 0.45814523100852966, 1.6157125234603882, 1.1221916675567627, -0.14998915791511536, 0.168503075838089, -1.416574478149414, 0.06529422849416733, -1.1270440816879272, -0.5808477997779846, -0.9415472745895386, 3.1774072647094727, -1.2701133489608765, -0.6467635035514832, 4.257839679718018, 0.6212913990020752, -1.112341284751892, -0.30806872248649597, -0.4142921566963196, 1.1266679763793945, -0.09363351762294769, 0.38435259461402893, 0.956580638885498, 2.0885744094848633, 0.18600249290466309, 1.0213279724121094, -0.21081143617630005, -0.6092585921287537, -0.9754433035850525, -0.09420077502727509, 0.7419958710670471, 0.5470086336135864, 0.7951430678367615, -1.6135692596435547, -2.2876675128936768, 1.538576364517212, 1.6072455644607544, 0.9156668782234192, -0.3901578187942505, 0.35996711254119873, 1.1901384592056274, 1.585127830505371, 0.9280382990837097, 0.07883787155151367, 0.6493456959724426, 0.04156578332185745, -0.6862905621528625, -0.903586208820343, 0.04205029457807541, 0.7834595441818237, -0.31664448976516724, 0.28335902094841003, -0.59999018907547, -0.06425336003303528, 0.685178816318512, 0.32405322790145874, 1.45198392868042, 0.0785975232720375, 0.7512460947036743, -0.46197015047073364, -0.4977310299873352, 0.1853639930486679, -0.5501063466072083, 2.030510663986206, 0.860643208026886, -0.39816832542419434, -0.502821147441864, 0.47033971548080444, 0.41802871227264404, -0.49142611026763916, -2.7810769081115723, -0.3855544924736023, -0.9757136106491089, -0.4825505316257477, 0.6848032474517822, 0.08184675872325897, 0.8058199882507324, -0.36998558044433594, -0.9214131236076355, 0.5146143436431885, -1.2942649126052856, 0.013944425620138645, 0.3714134693145752, 0.6106588244438171, -1.1854722499847412, -0.06186651438474655, -0.21530695259571075, -0.3463117480278015, 0.4903736114501953, 0.9609192609786987, -0.4195486903190613, -0.9321874976158142, 0.9244034886360168, -0.2703218460083008, -1.1746615171432495, 0.06319545954465866, -1.1799992322921753, -0.1284642219543457, -0.16713963449001312, -0.2583768963813782, 1.320550560951233, -0.6606975793838501, -0.8168186545372009, 2.2839393615722656, 0.1348479837179184, -0.21651695668697357, -0.012344686314463615, -1.014922857284546, -0.07432995736598969, -0.7153114676475525, 0.44374707341194153, -0.4026573598384857, -1.1341547966003418, -1.074804663658142, -0.46170300245285034, 0.9565155506134033, 0.10924272984266281, 1.0987544059753418, -0.2975369393825531, -1.0488442182540894, 0.09100005030632019, -1.3738378286361694, -0.4167201519012451, 1.1424155235290527, 1.6098902225494385, -0.7226609587669373, -0.8651669025421143, -0.7125710248947144, -0.1491287648677826, -0.6399464011192322, 1.313328742980957, -1.1068230867385864, -0.6789544224739075, -0.8230388164520264, -0.12002937495708466, 1.184915542602539, -1.062847375869751, 0.7358742952346802, -0.9752750992774963, 0.644474983215332, 0.03870485723018646, 0.036804284900426865, 0.08019402623176575, 0.27971044182777405, -0.32083043456077576, 1.2070140838623047, -1.0138633251190186, 0.8775467276573181, 0.12337339669466019, 0.6974270343780518, 0.8070454001426697, -0.6553746461868286, 0.5133717060089111, 0.28925466537475586, 0.4675549566745758, 0.4743489623069763, 0.15467149019241333, 2.059699773788452, 1.8472157716751099, -0.061854038387537, 1.0605823993682861, 1.2012474536895752, 0.3947424292564392, -1.273783564567566, 1.92942476272583, 1.0781364440917969, -0.014176253229379654, -0.6960425972938538, -0.42265287041664124, -0.153157576918602, 0.24054402112960815, 0.5744216442108154, 0.8748586177825928, 2.316560745239258, 0.29830431938171387, -0.5361629128456116, -0.45311424136161804, 0.6241840720176697, 1.2420462369918823, -1.319507360458374, 2.3794238567352295, 0.9429109692573547, 0.7393622398376465, 0.5529140830039978, -0.5288222432136536, -0.3618777096271515, 0.07441572099924088, 0.70274817943573, 0.73350989818573, 0.76811283826828, -0.987310528755188, -0.40825897455215454, -0.18636706471443176, 1.991858959197998, -1.5113989114761353, 0.5312926769256592, -1.0031561851501465, 1.8331199884414673, -0.7337363362312317, -0.51935875415802, -0.674324095249176, 0.7366310954093933, 0.22613200545310974, -0.7183430194854736, -0.8520290851593018, -0.8734022974967957, 1.2193559408187866, 0.4245564639568329, 1.2629362344741821, -0.41426098346710205, -0.4002929627895355, -0.29622936248779297, -0.3773215115070343, -1.2411534786224365, -0.4190322458744049, -1.9491050243377686, 0.2898160517215729, -0.16476763784885406, 0.7641629576683044, 1.7607086896896362, 1.1896376609802246, 1.6672663688659668, 0.8738592267036438, 0.6087812185287476, 0.35727038979530334, 1.6253736019134521, -0.9059721231460571, 0.2409159243106842, -1.2388275861740112, 1.2222239971160889, -0.5260015726089478, -0.44735008478164673, -0.44469892978668213, 0.014938694424927235, -0.5362153649330139, 0.4924807548522949, 1.2207590341567993, -1.0982811450958252, 2.0077664852142334, 0.18707020580768585, -0.7059279084205627, 0.23222938179969788, 0.640986442565918, 0.9746218323707581, -0.17593304812908173, 0.562020480632782, -1.0189979076385498, 0.46731680631637573, -0.4648696184158325, 1.0512564182281494, 2.6306872367858887, -1.8011082410812378, 0.9404553174972534, 2.5919535160064697, -0.35508406162261963, -0.9966258406639099, 0.6857988238334656, 0.08430614322423935, -0.44832944869995117, 0.9869470000267029, -0.24090643227100372, 1.081322193145752, 0.4129287302494049, 0.19629067182540894, -0.9623405933380127, -1.080068588256836, -0.7726576924324036, 0.8400182127952576, -0.5330679416656494, 0.9188823103904724, 0.5326932668685913, 0.7020658254623413, 1.9630271196365356, -0.8355763554573059, 0.38789045810699463, -0.24151214957237244, 1.1324052810668945, -0.6872137784957886, -2.142793893814087, -0.7219212651252747, 0.554847240447998, 0.2967333197593689, 0.9867925047874451, 1.0201270580291748, 1.293557047843933, 0.131582111120224, -1.1172815561294556, -0.5385369658470154, 0.6117790937423706, -1.5533126592636108, 0.5176717042922974, -0.537925660610199, 0.24457430839538574, 2.188660144805908, -0.9187824130058289, -0.5767845511436462, 1.5902612209320068, -0.027638055384159088, 0.4573917090892792, 0.2633163034915924, 1.6314302682876587, 0.6534567475318909, -0.013858454301953316, -1.221519947052002, 0.4454750418663025, -0.7650442123413086, -2.332648754119873, -2.32909893989563, -1.4732378721237183, 3.1265366077423096, 0.8854169845581055, 0.44463539123535156, 0.20526333153247833, -0.23818674683570862, -0.6963348388671875, -2.511427879333496, -0.27254945039749146, 0.6732121706008911, -0.643073320388794, 0.741048276424408, -0.05132973939180374, -0.6456751227378845, -1.55319082736969, -0.7764045000076294, -0.12830227613449097, -0.6333039999008179, -0.6708239316940308, -1.532469630241394, -1.4015756845474243, 1.1844912767410278, 0.2796265184879303, -0.33096519112586975, -0.33308184146881104, 0.8101810812950134, 1.9619656801223755, -0.9280548691749573, 0.22332479059696198, 1.4314548969268799, 0.3961017429828644, -1.3409960269927979, 1.8934191465377808, -1.129183053970337, -0.6410164833068848, 1.2654588222503662, -0.42994529008865356, 1.1760822534561157, 1.3995622396469116, -0.1321190595626831, -0.2166721075773239, 1.5503745079040527, -0.04269787669181824, -0.8779445886611938, -0.24469727277755737, 0.24013152718544006, -0.19690227508544922, 0.3014236092567444, 0.2899746894836426, -1.2082480192184448, -1.150575876235962, 0.09384485334157944, 1.0359036922454834, -2.64007830619812, -0.2124403864145279, 0.4628065526485443, 0.792521059513092, 0.4509470462799072, -0.964867115020752, 0.5650574564933777, -0.897634744644165, -0.9316245317459106, -0.7139193415641785, 1.2001889944076538, 0.6403471827507019, 1.338318943977356, 0.9107398986816406, 0.009844361804425716, -0.4680361747741699, -1.5692002773284912, -0.8996349573135376, 0.6297931671142578, 1.429675579071045, 0.3596414625644684, 1.9690872430801392, -0.45138686895370483, 2.037363052368164, 0.1655968725681305, 0.9728352427482605, -0.5659825801849365, -0.42459365725517273, 0.13146032392978668, 1.0114108324050903, 0.9285039305686951, 0.2747528851032257, -1.270925760269165, 0.7820793986320496, -0.2711885869503021, -0.6683964729309082, -1.9912325143814087, 1.6399739980697632, 1.78667151927948, -1.2931897640228271, 0.2589196264743805, -0.9457290768623352, -0.12630972266197205, -0.938639760017395, 0.9069986939430237, -0.18351292610168457, -0.09364358335733414, 0.4735959470272064, -1.0692049264907837, -3.395197629928589, 1.6272367238998413, 0.1988576054573059, 0.06970373541116714, -0.007665338460355997, -0.08809566497802734, -0.6363098621368408, -1.9479913711547852, -0.8892863392829895, -0.4334455132484436, -0.24930110573768616, 0.4677428901195526, 1.1333260536193848, 0.0030489435885101557, 0.6141774654388428, 1.3459640741348267, 1.9931697845458984, -0.5704247355461121, -0.09290624409914017, 0.7323977947235107, -1.5533496141433716, -0.7336300611495972, -0.34998977184295654, -1.2082806825637817, 1.0708967447280884, 0.5412386059761047, 2.624101161956787, -0.222259059548378, 1.0619604587554932, 0.28275221586227417, 1.4894386529922485, 1.623511552810669, -0.5637363195419312, -0.8724271655082703, -0.017692118883132935, -0.45486000180244446, 0.4899294674396515, 0.3493064045906067, 0.5273771286010742, -1.5149821043014526, 0.6181467175483704, -0.406425416469574, -0.9625725150108337, 0.5235268473625183, 1.2552481889724731, -0.3629390001296997, 0.8163197040557861, 0.4788668155670166, 0.4619988203048706, 0.5924375653266907, -0.7270429730415344, -0.905137300491333, 1.2461460828781128, 0.4989534020423889, 0.863262414932251, -1.5936192274093628, -0.34616902470588684, 1.7913187742233276, 1.5510376691818237, 0.5975766777992249, -0.30497971177101135, -0.8734748959541321, -0.45643457770347595, -1.531055212020874, -0.9067763090133667, -0.943392276763916, 1.1375398635864258, -0.11666105687618256, 0.6325346231460571, -0.48983898758888245, 0.9689608812332153, -1.2696106433868408, -0.7425653338432312, 0.9269241690635681, -0.7295953035354614, -0.22391724586486816, -0.039194341748952866, 0.8393356800079346, -0.28804120421409607, -1.417069911956787, 0.8069683909416199, 1.6160939931869507, -1.3560950756072998, 1.5090441703796387, 1.9462610483169556, -1.2815577983856201, 1.2886013984680176, 0.7377892136573792, -0.022320128977298737, -0.058902934193611145, 0.35538119077682495, -0.7816171646118164, -0.13018013536930084, -0.925816535949707, 1.5743094682693481, 0.12520672380924225, -0.4748815894126892, 0.3933946490287781, 0.05112317577004433, 0.12331537157297134, -0.8780835270881653, -0.04107826575636864, 0.008180099539458752, 1.7802672386169434, -0.22317831218242645, -1.7914382219314575, -1.1922295093536377, -0.6856727004051208, 0.5339328050613403, 1.6013532876968384, 0.3681390583515167, 1.8766344785690308, -0.7330105304718018, -0.01905435137450695, 1.221674919128418, 0.17384934425354004, 0.9394868612289429, 1.336610198020935, -0.3093178868293762, -0.14550696313381195, -0.6003822088241577, -1.51797354221344, 1.7580569982528687, -1.355633020401001, 0.7524902820587158, -0.7157060503959656, -0.37026652693748474, -0.22529111802577972, 0.5581435561180115, -0.6970020532608032, -1.2161859273910522, 0.6132988929748535, -0.6373251080513, 0.676649272441864, -0.8422946929931641, -0.6177936792373657, -1.3551760911941528, -1.1669448614120483, 1.0481600761413574, -0.15823273360729218, 0.45459049940109253, 0.7145344018936157, -0.4750377833843231, -0.2798474133014679, 0.40258750319480896, 0.3969065845012665, 2.985816478729248, 0.14552204310894012, -0.2440616935491562, -1.8026511669158936, -1.2845138311386108, 0.809235692024231, -0.31021466851234436, -0.7109576463699341, 0.6194107532501221, 0.9598940014839172, -1.2090063095092773, -0.5639064311981201, 0.7374390959739685, 0.8340238332748413, -0.2428838014602661, -0.28008607029914856, -0.17034921050071716, 0.8454234004020691, -1.1697739362716675, 2.1124656200408936, 0.8790879249572754, -1.674418568611145, -0.048430245369672775, 0.6987466216087341, -0.3435276746749878, 0.2040117383003235, -2.0204696655273438, 1.785894751548767, -0.8638964295387268, -0.6310527324676514, -1.9881259202957153, 0.7018157839775085, -0.5360447764396667, -0.17685940861701965, 1.6751576662063599, -1.7031887769699097, -0.05315866693854332, -1.8639625310897827, 0.6276443600654602, 0.24576318264007568, 0.08536053448915482, 0.23112894594669342, -0.6405606865882874, 0.28444862365722656, 0.1696745753288269, -0.214682474732399, 0.8593350052833557, 0.5303031206130981, -0.3146214187145233, 0.28398486971855164, 1.1196129322052002, 1.23061203956604, -0.03339853510260582, -0.12611469626426697, 0.12735502421855927, 0.9101876020431519, 0.44572168588638306, -0.4181070029735565, -0.5937047004699707, 0.3209819197654724, 0.7689050436019897, 0.9065008759498596, -0.9795956611633301, 0.31469497084617615, -0.5388802289962769, 1.4210835695266724, 0.4119642972946167, 0.03558953106403351, -0.37590715289115906, 0.4309583008289337, -0.22852635383605957, 0.9219990372657776, -1.1534982919692993, -0.8567194938659668, 0.24184755980968475, -0.12766826152801514, -2.103489637374878, 0.33477240800857544, -1.1273703575134277, -0.2019350677728653, -0.8006521463394165, 2.62272047996521, -0.496507853269577, 0.9436353445053101, -0.06762032955884933, -0.29542937874794006, 0.9329994320869446, -1.4577621221542358, -2.0781009197235107, -0.17441123723983765, -0.9694159030914307, -1.2936557531356812, -0.8135282397270203, 0.03361538052558899, -0.048027440905570984, -1.098435401916504, -0.8309897780418396, 0.2247486710548401, 0.5544382333755493, 0.31205224990844727, 0.92598557472229, -1.6453145742416382, 1.1588149070739746, -1.3481932878494263, -1.4063036441802979, 0.4059574604034424, 1.505635380744934, -0.9965190887451172, -0.38007989525794983, -0.6195290088653564, -0.48865389823913574, 0.9207331538200378, -1.0014575719833374, -0.011486752890050411, -0.2745473086833954, -1.0401866436004639, 0.04248423874378204, 0.7517949938774109, 0.998779833316803, 0.4773043990135193, -1.5012257099151611, -1.391667127609253, -0.8871563673019409, -0.9157143235206604, 0.17416663467884064, -1.438354730606079, 1.0475704669952393, 1.981866478919983, 0.2331932634115219, 0.942511796951294, 0.4363645315170288, -1.9447249174118042, 1.5135561227798462, 0.977652907371521, -1.652790904045105, 0.060454629361629486, 0.6062986850738525, 0.3119995892047882, 2.6745550632476807, 1.0136630535125732, 0.09825151413679123, -2.2781920433044434, -1.6587281227111816, 2.000523567199707, -1.3816975355148315, -1.2803555727005005, 1.3524422645568848, -0.3011096119880676, -1.0823367834091187, 0.4423176050186157, 0.22790570557117462, 0.39821168780326843, -0.7831573486328125, 0.2230139523744583, 1.1881632804870605, 0.24986565113067627, -0.6832637786865234, 1.2333357334136963, -0.12472468614578247, -1.3962609767913818, 0.6672907471656799, -0.5567235350608826, -0.7908787131309509, -0.060827914625406265, 0.9902598261833191, -1.266032338142395, 1.5936684608459473, 1.7354522943496704, 0.6852678656578064, 0.12054178863763809, -1.356598973274231, 0.3319290280342102, 1.262870192527771, -0.736725389957428, 0.22110064327716827, 0.11701854318380356, 0.6801027059555054, -0.3649354577064514, -0.3561873137950897, 1.5496280193328857, -0.08075567334890366, 0.7093316316604614, -0.6220932006835938, 0.5488158464431763, -0.8555305004119873, -0.3445286452770233, -0.816017210483551, -0.6389387845993042, -0.05474807322025299, 0.2607176601886749, -0.4320998191833496, -0.09950681030750275, 1.051615595817566, -0.21599872410297394, 1.5730398893356323, -0.4087182879447937, 1.1885405778884888, -0.15122829377651215, 0.6821707487106323, 0.058947280049324036, 0.7001920342445374, 0.6324625015258789, -2.2399537563323975, -0.8853283524513245, -0.08177926391363144, 2.4197683334350586, 0.10523594915866852, 1.5375972986221313, -1.2640743255615234, -0.781857967376709, 1.4013453722000122, 0.49330461025238037, -0.10709994286298752, -0.6925469040870667, -0.3965623378753662, -0.5376201272010803, 0.41098350286483765, -0.04208892956376076, -0.4530624747276306, 0.20295275747776031, 0.856891393661499, -1.6742572784423828, 1.5206083059310913, 0.6425341367721558, 0.5236414074897766, -1.0189988613128662, 1.0181251764297485, -0.467130571603775, 1.053431510925293, -0.7996605634689331, -0.46230506896972656, 0.26272839307785034, 0.007215912453830242, 0.16743101179599762, -1.2058697938919067, -0.5651443600654602, -1.0138496160507202, -1.6394034624099731, 0.6240007877349854, 1.2441699504852295, 0.4256540834903717, -0.8636308312416077, -0.6596384644508362, -0.24771738052368164, -0.410736620426178, 1.7754088640213013, 0.022073274478316307, -0.923168420791626, 0.6969621777534485, -0.3660788834095001, -0.18686839938163757, -0.24229282140731812, 1.3533015251159668, 0.4133097231388092, 0.9490817785263062, -0.45844608545303345, 0.2302813082933426, -0.9187622666358948, -0.9489753842353821, -1.9109774827957153, 2.0298213958740234, -0.9256835579872131, -0.8112816214561462, -1.4697914123535156, 1.2544445991516113, -1.0219347476959229, -0.5335063338279724, -1.2637351751327515, 0.4182616174221039, 0.8876612782478333, -0.8890355229377747, 0.7419907450675964, 0.3599121868610382, 0.7647928595542908, -0.6331834197044373, -1.5353891849517822, 0.4276929795742035, 0.7122746109962463, -2.0463497638702393, 0.6341784596443176, 0.11058937013149261, 1.419702410697937, 0.8503522872924805, -0.5022768974304199, -0.6114704012870789, -1.740088939666748, -0.9237504601478577, -0.5639649033546448, -0.007366002071648836, 1.410352110862732, 1.4958330392837524, -1.3024046421051025, 0.9814668297767639, 0.6454346179962158, -0.11881644278764725, 0.6260537505149841, -0.9552068114280701, 0.43327596783638, 0.33430445194244385, 1.5044246912002563, 0.7209390997886658, 0.4635342061519623, 0.7390735745429993, 0.4228573441505432, -1.1147507429122925, -1.0860480070114136, -0.5174466967582703, -1.0537270307540894, -0.8304483890533447, 0.6440595388412476, 2.17990779876709, 0.24404066801071167, -1.1516830921173096, -0.6037436127662659, -1.3695918321609497, 0.6169762015342712, -0.8388493061065674, 1.2318068742752075, -0.055126022547483444, 1.5634779930114746, 1.7047061920166016, -0.1333896666765213, 0.7671509981155396, -0.9616166353225708, -0.5329949259757996, -0.4805815517902374, 0.34807315468788147, -0.9070920348167419, -2.880159378051758, 0.045077867805957794, 0.1910955309867859, 0.2158605307340622, 0.19650059938430786, -0.5619637370109558, 2.87028169631958, -1.2779994010925293, -0.8028307557106018, 1.8486210107803345, 0.4686131477355957, -1.039940595626831, 1.130544662475586, -0.7402728796005249, -0.02314348705112934, 1.3281869888305664, 0.9818559288978577, 2.221606969833374, 0.8495923280715942, -1.1475213766098022, -0.0240156352519989, -0.2729862630367279, 0.4092855155467987, 0.010729150846600533, -0.9448139667510986, 0.7617403864860535, -0.18764802813529968, -0.00877284910529852, -0.48104622960090637, -0.2544226050376892, -2.236013650894165, -0.8024281859397888, -0.9762537479400635, 0.47382092475891113, 0.5344647169113159, 0.4155259132385254, 0.1789465695619583, -1.6293559074401855, 1.4373259544372559, -0.4090506136417389, 0.38414183259010315, -0.10275101661682129, 0.5303990244865417, 0.5509370565414429, 0.7763851284980774, -0.36744654178619385, -2.4345455169677734, -1.5691834688186646, -0.829355001449585, 0.5281957387924194, 0.8287611603736877, 2.032439947128296, -0.4313162565231323, -1.329114317893982, -0.31775474548339844, 0.20679871737957, 1.0263019800186157, 0.615265965461731, -0.08468881249427795, 0.9008669257164001, 0.7002174854278564, 0.6129715442657471, 0.34466254711151123, -0.12053193151950836, -1.2666304111480713, -0.701213002204895, 0.946540355682373, 0.3373974859714508, 0.7440884113311768, -0.7756665349006653, 0.43425050377845764, -0.08657783269882202, 0.39663079380989075, -0.9757428169250488, 0.9878358840942383, -0.20628227293491364, -0.014655409380793571, 0.3759204149246216, 0.36350780725479126, -0.3779616057872772, -0.5426444411277771, 2.0818850994110107, 0.5117878913879395, -0.8357879519462585, 0.06577261537313461, 0.9088199138641357, 2.1205852031707764, -0.23876380920410156], [1.4468475580215454, 1.3265914916992188, 1.0820975303649902, -0.8976753950119019, -0.17921586334705353, 0.03350786119699478, 0.7211480140686035, 0.1450711041688919, -0.5863618850708008, -0.2519686222076416, 0.02034311182796955, 0.1211637482047081, 1.3024144172668457, 0.7439313530921936, 1.3148666620254517, -0.9639713168144226, -1.3237143754959106, -0.12265291810035706, -1.6294364929199219, -0.2847086191177368, -0.30371612310409546, 1.0086311101913452, -0.9100273251533508, -0.7771633863449097, 2.3293867111206055, 0.8914889097213745, -0.6075920462608337, -0.6743074655532837, 0.22555777430534363, -0.5937865376472473, 0.5192515850067139, -0.5085352659225464, 0.06970055401325226, 1.2669090032577515, -0.6507228016853333, -1.6615827083587646, 0.5092650651931763, -0.9562361240386963, 0.20439991354942322, -0.43574902415275574, -0.36365750432014465, -1.339016318321228, -0.32686954736709595, 1.3999353647232056, -0.45759692788124084, -0.8234843015670776, 1.9387421607971191, -1.271563172340393, 0.36344456672668457, 0.4110005795955658, 0.9154902100563049, -0.8808769583702087, -0.15608537197113037, 0.2323916107416153, 0.4527428150177002, -0.09515159577131271, -0.8873559832572937, -1.2854355573654175, -1.301701307296753, -1.0441597700119019, 0.707600474357605, 1.216251254081726, -1.7167956829071045, -0.4746115803718567, 0.7293534874916077, 0.3299461007118225, -1.575905680656433, -0.8457844853401184, -0.6361562013626099, -1.2843235731124878, -0.5445550084114075, 1.5739675760269165, 0.05668191239237785, 1.7058378458023071, -0.8780891299247742, -1.3448177576065063, -1.5738606452941895, 0.17936396598815918, 0.6669507026672363, 1.526261329650879, -0.5935583710670471, -0.08355998992919922, -2.8171546459198, 0.6688292026519775, 0.5239916443824768, -0.4850020110607147, -1.0457197427749634, -0.5152933597564697, 0.9488425850868225, -1.2889020442962646, -0.7251205444335938, -0.07804642617702484, 0.8490452170372009, 1.5439831018447876, 1.3066900968551636, -1.1789485216140747, -0.09620220959186554, -0.532301664352417, 1.9514309167861938, -0.8655994534492493, 0.030193736776709557, -0.4743940234184265, -0.26657065749168396, 0.3614999055862427, 0.17416471242904663, -0.3996649384498596, 1.321484088897705, 1.1446791887283325, -0.34213146567344666, -1.594796061515808, -0.4720782935619354, -0.19265152513980865, 0.3059203624725342, -0.5587654113769531, -0.25205016136169434, -1.0266821384429932, -0.13461603224277496, -0.5626561045646667, 0.39454078674316406, 0.3996095061302185, 2.1573598384857178, 0.5442504286766052, 0.33523520827293396, 1.063164234161377, 0.9345335960388184, -0.12375615537166595, -0.2806102931499481, 1.599227786064148, -0.06309546530246735, 1.361669898033142, 1.0876885652542114, -0.6240442395210266, -0.9063519835472107, 0.11614775657653809, -1.5871628522872925, -0.5626376867294312, -1.4555414915084839, -1.560713768005371, -0.5162518620491028, 0.9971428513526917, 0.018301067873835564, -1.7807157039642334, 0.10553004592657089, 1.5030597448349, -0.8990205526351929, 0.15481232106685638, -0.6807401180267334, 0.834568440914154, -0.42321527004241943, -1.0321911573410034, -0.7525985240936279, 0.6902844309806824, -0.4084361791610718, 0.3589441180229187, -0.8636231422424316, 0.08651579171419144, 0.5469827055931091, 0.4348090589046478, 1.0768773555755615, 0.2478993535041809, -0.45946329832077026, 0.1369471549987793, 0.582209587097168, 0.0665220096707344, 0.19048622250556946, 0.33156299591064453, 0.18660300970077515, -0.20697790384292603, -0.8177523016929626, -0.12171100825071335, -1.1925370693206787, 1.097304344177246, -0.1982804387807846, -0.5515520572662354, -0.5602103471755981, -1.6697219610214233, 0.40476638078689575, 1.5369826555252075, -1.7128793001174927, 0.11568716913461685, -0.13715124130249023, -0.3546426594257355, 0.7587394118309021, 0.1909925937652588, 0.1756901890039444, 1.5310231447219849, 1.6148399114608765, 0.27635931968688965, -0.6855493187904358, 0.6187567114830017, 0.3304925858974457, -0.9768690466880798, -1.5744552612304688, -0.6100181341171265, 2.154191255569458, -1.0132919549942017, 1.4728306531906128, 1.13554048538208, -0.022568123415112495, -1.8464444875717163, 2.2696030139923096, 0.5650804042816162, -0.186833456158638, 0.9918463826179504, -0.4464969336986542, -0.06363904476165771, -0.921688973903656, 0.38720646500587463, -0.15508782863616943, 1.3341456651687622, 0.8009611964225769, 1.179460048675537, -0.3791702091693878, 0.08529429137706757, 0.6269845962524414, -2.046856164932251, -0.17041119933128357, 0.5368342399597168, 1.2005537748336792, -1.3160607814788818, 0.1774069219827652, 1.3521153926849365, 0.3291057050228119, -2.123461961746216, 0.39854034781455994, -1.5677385330200195, -0.5129081010818481, -0.8022148013114929, -0.7389711737632751, 0.8356406688690186, 1.3705799579620361, -0.34287822246551514, -0.31752586364746094, 1.6764614582061768, -0.8719311952590942, 0.8064224720001221, -1.0750224590301514, 0.7475062608718872, -0.56746506690979, -1.8818004131317139, -1.3779802322387695, 0.5094292759895325, -1.4873634576797485, 0.34577858448028564, -1.7883813381195068, -0.32559069991111755, -1.8551647663116455, -0.9554555416107178, 0.08755763620138168, 1.2358832359313965, 1.3318382501602173, 1.3725240230560303, 2.099591016769409, 0.7529076933860779, 0.1972191035747528, -1.2520649433135986, 0.5045121908187866, -0.4453602135181427, 1.189185380935669, 1.994396686553955, -0.28146445751190186, -1.7492140531539917, -1.0415161848068237, 0.10086694359779358, -0.03261944651603699, -1.1026737689971924, 1.040346384048462, 0.10220491141080856, -0.0555475689470768, -0.17340371012687683, -1.0109368562698364, 0.015080570243299007, 1.9737229347229004, 1.3028087615966797, 0.5312371253967285, 1.8197554349899292, -0.04483675956726074, 0.3197186589241028, -0.504538357257843, 0.4362020492553711, 1.4814965724945068, -1.6547539234161377, 0.18091875314712524, 0.7022125124931335, -1.1677377223968506, 0.30164194107055664, 0.0773371160030365, -1.0163480043411255, -0.991818368434906, 1.1548877954483032, -1.0988223552703857, 0.21913720667362213, 0.2071431279182434, -1.4295586347579956, -2.0650432109832764, -0.8381418585777283, 0.7469854950904846, 0.09543371945619583, -0.053352415561676025, 0.43630507588386536, 0.15504898130893707, 0.25979098677635193, -0.1357550173997879, 0.5219634175300598, 0.4152439534664154, 0.7297540903091431, 0.8668882250785828, 0.0026456047780811787, 1.5349560976028442, 0.26967647671699524, -0.4802801012992859, -3.309431314468384, 2.2269372940063477, -0.6322380900382996, 0.0668153166770935, 0.43839848041534424, 1.6185814142227173, 1.0542770624160767, 0.6002108454704285, 1.1333340406417847, 0.08961088955402374, 0.7228243947029114, -1.0025488138198853, 0.16379708051681519, -0.8598816990852356, 2.3319950103759766, -0.5473666787147522, 1.3510435819625854, -1.0017690658569336, -1.566439151763916, 1.6585341691970825, 0.3013184070587158, -1.014654278755188, -1.4776687622070312, -0.21403853595256805, 0.09225773066282272, 0.7797115445137024, -1.6861597299575806, 0.5452449321746826, -0.34827956557273865, -0.861017107963562, -0.7858117818832397, -0.013149535283446312, -0.027508018538355827, 0.0740184336900711, 1.0467056035995483, -1.8832014799118042, 0.9813669919967651, 0.5796509981155396, -0.6399979591369629, -0.4461027681827545, 0.2661401629447937, 2.6852970123291016, 0.0517074316740036, -2.1427829265594482, 0.3933972418308258, 1.043110728263855, 0.896426796913147, -1.33810555934906, 0.17854805290699005, 1.2644098997116089, 0.8719085454940796, -0.04655289277434349, -0.14102455973625183, 0.7000828981399536, -0.240533709526062, -1.0004279613494873, -1.0667352676391602, 1.7556558847427368, 0.6380388736724854, 1.0141217708587646, -0.1860102266073227, -0.05983173847198486, -1.620537281036377, 0.36594000458717346, 0.1632976084947586, 0.5286670923233032, -0.4376561939716339, -0.6164464950561523, 1.000059723854065, -0.47333797812461853, -0.33364343643188477, -0.7617970705032349, 0.3800938129425049, 0.8187457919120789, 0.2578267753124237, 0.07518547773361206, -2.592705011367798, 0.8313487768173218, -1.5938973426818848, -0.0429069958627224, -1.2158071994781494, 1.7438329458236694, 0.6811761856079102, 0.8707252144813538, 0.3934456706047058, 0.5640560388565063, 2.1588330268859863, 0.15241406857967377, -2.168848752975464, -0.7521275877952576, 1.4821807146072388, -0.5643840432167053, -0.652601957321167, 1.2576165199279785, -0.655493438243866, -0.8362531661987305, -0.20508986711502075, -1.226675271987915, -0.1906408816576004, -1.3181241750717163, 0.603165328502655, 0.9801180958747864, 1.1255797147750854, 0.8264977335929871, -0.4642482101917267, -0.8732990622520447, -0.9288597702980042, 0.7131091356277466, -0.12220972031354904, -0.6233327388763428, 0.2644810676574707, -0.742795467376709, 0.22677026689052582, 0.30682557821273804, 0.8999171853065491, 1.0395034551620483, -1.5613820552825928, -0.6896781325340271, -0.2404390275478363, -1.4786988496780396, 1.0395433902740479, 0.7512787580490112, 0.06672373414039612, 0.889167308807373, -0.8828967809677124, -0.028059186413884163, 0.6905317902565002, -0.4991641044616699, -1.3107324838638306, 0.39118003845214844, -1.418511986732483, -0.06686405837535858, 0.4938164949417114, 0.5007630586624146, 0.4975953698158264, -0.06020829826593399, 1.6593568325042725, 0.3941931128501892, -0.5880029201507568, -0.40275585651397705, 0.39993178844451904, -1.367567539215088, -0.37834250926971436, 0.9528520703315735, 1.0851998329162598, -0.8543130159378052, -0.19741596281528473, 0.9878225326538086, 0.9679949283599854, -1.7410565614700317, 0.05167122185230255, 0.2679753005504608, -0.04645046219229698, 0.7559519410133362, -0.5073740482330322, -0.33457133173942566, 0.3119586408138275, 0.7727100253105164, 0.472552090883255, -1.1822696924209595, -0.8019934296607971, 0.2753294110298157, 0.299724280834198, -0.6759474873542786, 0.6234000325202942, 2.068631410598755, -0.7454177737236023, -0.07470755279064178, 1.1097537279129028, -0.5043527483940125, -0.09276602417230606, -0.2524758577346802, -0.2223977893590927, -0.2816344201564789, -0.2956559956073761, -0.4393286108970642, -0.478741317987442, -0.18786630034446716, -2.8348827362060547, -2.736570119857788, -0.3981661796569824, -0.15918612480163574, 1.0614527463912964, 0.20441056787967682, 0.45669782161712646, -1.3394509553909302, 0.4521593451499939, -0.7919237613677979, -1.8626376390457153, 1.1345007419586182, 2.4070656299591064, 1.4354997873306274, 0.3336911201477051, -1.9267584085464478, -0.009842956438660622, 0.7742690443992615, -0.6633962988853455, -0.3839489221572876, 1.0422208309173584, -0.48470908403396606, -1.9756524562835693, 1.032098412513733, 0.010730020701885223, -0.28106287121772766, 0.3006186783313751, -0.31229206919670105, -0.25745201110839844, 0.5507532954216003, -0.9752522706985474, -1.5274475812911987, 0.1583561897277832, -0.9939449429512024, -0.6028450131416321, 0.12258275598287582, -0.8531689047813416, 1.4476932287216187, -0.30878084897994995, -0.13829980790615082, -1.3533217906951904, -1.0992321968078613, 1.7924612760543823, 0.5242661237716675, -0.7248296737670898, -0.4007166028022766, -0.7600876688957214, -0.4102623760700226, -1.1419646739959717, -0.738808274269104, 0.23043930530548096, -0.28748205304145813, -0.8919046521186829, 0.09096793085336685, -2.827402114868164, -0.46217164397239685, 0.927428126335144, 0.4891510307788849, -0.19953857362270355, 0.03239328786730766, -0.05051719769835472, 0.08456499129533768, 0.40699559450149536, -0.28884679079055786, 1.4712179899215698, 0.12598904967308044, -1.4245941638946533, 0.8547593951225281, -2.2957351207733154, -0.8257618546485901, 0.6369138956069946, -2.036367416381836, -0.7375049591064453, 0.7254294157028198, 0.7370645403862, 0.00772578502073884, 0.487314373254776, 1.0326510667800903, 0.5503866076469421, 0.28782638907432556, 0.6356212496757507, -0.7823452949523926, -0.0028138398192822933, -0.5252918601036072, -0.5638293027877808, -0.18835358321666718, -1.6200922727584839, -0.7515903115272522, 0.09158995747566223, -0.7504382729530334, -0.9065848588943481, 0.6684069037437439, 0.024676833301782608, 0.7076785564422607, 0.5193158388137817, 2.2947986125946045, -0.21427194774150848, 0.21153821051120758, -0.1563909500837326, 0.06109046936035156, -0.8468934297561646, 0.31082409620285034, -1.2441632747650146, 0.1675885021686554, 0.2933783233165741, 1.6616508960723877, -2.196925163269043, 0.3809307813644409, -1.7782025337219238, -0.08360883593559265, 0.5431271195411682, -0.280073881149292, -0.579208493232727, 0.0007196870865300298, -0.9297906756401062, 2.121598720550537, 1.4406846761703491, 1.2098668813705444, -0.363863468170166, 1.7100439071655273, 1.0519328117370605, 0.754589319229126, -0.8487591743469238, 1.1313382387161255, -0.8797848224639893, 2.0345206260681152, -0.6460012793540955, -0.8895748257637024, 1.3120509386062622, 0.4696114957332611, -0.6373433470726013, -0.022848084568977356, -0.8536013960838318, 0.24989035725593567, 0.543765127658844, 0.9899270534515381, -0.47946667671203613, -0.39294278621673584, 0.8337574601173401, -0.8273134231567383, 0.7040712833404541, 0.23895679414272308, 0.0456082783639431, 1.4564523696899414, -1.050094723701477, -0.07699298858642578, 1.673542857170105, -0.46880847215652466, 0.4784903824329376, -0.6039045453071594, 0.9994248151779175, 0.21372701227664948, -0.10261859744787216, 0.21805880963802338, -0.4155609905719757, -0.019570542499423027, 0.19363386929035187, 1.257256031036377, -1.9500114917755127, 1.608269453048706, -0.9038742184638977, -1.0578248500823975, 1.058140516281128, 1.5636680126190186, 0.11842945963144302, -0.45721006393432617, 0.6950874328613281, 0.4765945076942444, 0.36964845657348633, 0.3886334002017975, -0.6857439279556274, -0.903156578540802, 0.135712668299675, -0.7514152526855469, -1.119788408279419, -0.9876702427864075, 0.5736646056175232, -1.8401597738265991, 0.4275895059108734, -2.509110689163208, -1.8299593925476074, -0.04167260229587555, -0.3766706585884094, -0.1890692412853241, 1.110883355140686, 0.9354589581489563, -0.04317063093185425, 0.4964480996131897, 0.42541274428367615, -1.1318522691726685, 0.7012220621109009, -1.3271315097808838, 1.05315101146698, -0.37558653950691223, -1.121322512626648, 1.6039668321609497, 0.27036330103874207, 0.14837852120399475, 0.2902536988258362, 0.737623929977417, 1.5520325899124146, -0.7766412496566772, -0.6186938881874084, -1.9828766584396362, -0.6190892457962036, -1.073842167854309, -1.2285219430923462, -0.36264270544052124, 1.690927505493164, 0.1244262307882309, -0.13470511138439178, 0.7096765041351318, 0.12269257009029388, -0.5990455150604248, -0.2519942820072174, 0.6025910377502441, 0.8733962774276733, -1.6866438388824463, -0.2362157255411148, 1.6964222192764282, -0.7140678763389587, 0.7633309960365295, -1.6866936683654785, 0.4725348949432373, -0.43244630098342896, -0.7715930938720703, 0.1373007446527481, -0.5711971521377563, -0.015443081967532635, -2.2043159008026123, 0.8787867426872253, 0.09788828343153, 1.2532439231872559, -1.0417121648788452, 0.05060330778360367, -1.1671162843704224, -0.4169718623161316, 1.3121002912521362, 0.5625402331352234, 0.8812090158462524, -0.12506631016731262, -1.753102421760559, 0.12077753990888596, 2.461904764175415, -0.5384643077850342, -0.17239436507225037, 0.29857608675956726, 0.2568756937980652, 1.883293628692627, -0.7181383371353149, -1.470092535018921, 0.46295616030693054, 0.17105552554130554, -0.05071159079670906, 0.5543695688247681, -0.7609311938285828, 0.266141414642334, 1.8951870203018188, -0.6078492403030396, -0.7014084458351135, -0.804276168346405, 1.4741655588150024, -0.2660374343395233, -1.1605761051177979, 0.6065374612808228, -2.016641616821289, 0.8023331165313721, -0.7200331091880798, -0.5784949064254761, -0.2899772524833679, 2.1634979248046875, -0.0007393572013825178, 0.01830085925757885, -0.08536989241838455, -1.4966444969177246, 0.37466225028038025, -1.0377553701400757, -0.5112413167953491, -0.4118022918701172, -1.2309339046478271, -0.37998539209365845, -1.1549475193023682, 0.3848663568496704, 0.5923746824264526, 1.1093592643737793, -2.221360921859741, -0.19258229434490204, -0.623466968536377, 0.39130958914756775, 0.3127114474773407, 1.638912558555603, 0.7343675494194031, -0.17273759841918945, 0.9609079957008362, -0.9573948979377747, 0.593640923500061, -0.255685418844223, 0.4838188588619232, 1.6317665576934814, -1.2717303037643433, 1.5948244333267212, 1.2884169816970825, 0.550178587436676, -0.8091039061546326, -1.0229699611663818, 0.8875079154968262, -0.452134907245636, 0.9350486397743225, -1.098431944847107, 0.3290849030017853, 0.8834616541862488, -2.273642063140869, 1.1849145889282227, -0.5513011813163757, -1.5832008123397827, 1.5471798181533813, -1.2662943601608276, -0.5112999081611633, 0.5520324110984802, -0.6138821244239807, -0.34413960576057434, -0.741898775100708, 0.3419727087020874, 1.3151367902755737, -0.18508006632328033, 0.013516947627067566, -0.5510571599006653, 0.3401181399822235, -3.273745059967041, -0.32237496972084045, 0.16070963442325592, -0.6908116936683655, 1.015067219734192, -0.37375718355178833, -0.7968626618385315, 0.34896302223205566, -0.416024386882782, 0.25205594301223755, -0.9776811599731445, -0.013169759884476662, -0.4448893964290619, 0.1267295628786087, 1.5254600048065186, -0.7714449167251587, 1.1051348447799683, 0.7645792365074158, 0.9804127812385559, -0.9263597726821899, -1.655148983001709, -1.0122287273406982, -0.33179426193237305, -1.1639411449432373, 1.113250494003296, -0.2758922576904297, 1.61995267868042, 1.6323946714401245, 0.546898603439331, -1.1603745222091675, 0.8969685435295105, -0.8388164639472961, 1.056077480316162, -1.1087968349456787, 0.24900703132152557, 0.39592432975769043, 1.541107416152954, -2.071444511413574, 1.220702886581421, -0.12536878883838654, -0.2596336901187897, -1.534277081489563, -0.9905654788017273, -0.8916222453117371, 1.199240803718567, 0.46856826543807983, -1.2809174060821533, 1.7567344903945923, 0.06485207378864288, -1.126416802406311, -0.11310429126024246, -0.31956812739372253, 0.8798039555549622, -0.7035039663314819, 1.7158814668655396, -1.2288408279418945, 1.8641605377197266, -0.5094325542449951, -0.7459961175918579, -0.43752506375312805, 1.2174630165100098, 1.3218094110488892, -0.5399098992347717, 1.2394514083862305, -1.5042308568954468, -0.567201554775238, 0.5835314989089966, 0.26206231117248535, -0.25632214546203613, -0.9735673666000366, -1.1220922470092773, 0.29150503873825073, -0.3589169979095459, -0.07315677404403687, 0.14557400345802307, -1.8526332378387451, 0.7223911285400391, 0.3717154562473297, -1.7334072589874268, 0.9173626899719238, 0.8647854328155518, -0.1592627763748169, 0.3057512044906616, 0.1963805854320526, -0.5294713973999023, 0.5555081367492676, 0.9561376571655273, 0.5639724135398865, -0.18026429414749146, 0.673903226852417, 0.7089584469795227, -0.6993446946144104, -0.5870141386985779, 0.1572585254907608, -0.49902477860450745, 0.1910056173801422, -0.5211946964263916, -0.6866708993911743, -0.04928627982735634, -0.09557385742664337, 0.13807202875614166, -1.396340250968933, -0.1444738805294037, 0.19980739057064056, 0.4898228049278259, -0.09139902889728546, -0.5955246686935425, 0.3480031490325928, -0.47272688150405884, 2.0475378036499023, 2.8383736610412598, -1.4104918241500854, -1.1939233541488647, 0.6220734119415283, -0.9344375133514404, 1.9476722478866577, 0.3134538531303406, 0.10901568830013275, -1.9048727750778198, 0.9052448868751526, 0.005204471293836832, 1.1350345611572266, 0.4947185814380646, 0.0660017654299736, -1.6644612550735474, -0.34599000215530396, 0.5062116980552673, 1.1227947473526, -0.5719197988510132, -0.10472548007965088, -0.3459426462650299, -0.8324835896492004, 0.47707197070121765, -1.6210392713546753, 0.6713301539421082, -2.1866867542266846, -0.24092409014701843, -1.1728136539459229, -0.18405212461948395, -0.18159015476703644, 0.9560226202011108, 0.24471242725849152, 1.5793880224227905, -0.4328087270259857, -0.2658288776874542, 0.37280964851379395, 0.35157617926597595, -1.4229453802108765, 1.7162436246871948, 0.6773115396499634, 0.20928803086280823, -0.09886616468429565, -0.01596619375050068, 0.06647099554538727, 1.0945111513137817, -0.4920019507408142, -0.15622872114181519, 1.9555214643478394, 0.10536786913871765, -0.5305212736129761, -1.547028660774231, 0.6737422347068787, -1.088591456413269, 0.565038800239563, 1.1826071739196777, -0.21750229597091675, -0.7671337127685547, 1.046992540359497, 0.0010393343400210142, -0.9182136058807373, -1.1453807353973389, 0.42662569880485535, -1.1571948528289795, 0.14121262729167938, -0.4763539135456085, -0.42405015230178833, -0.38215532898902893, 0.5287078619003296, -0.43374213576316833, 0.6840903162956238, 1.5964829921722412, 0.19366858899593353, 2.058396816253662, -0.6444534659385681, -0.9930834770202637, 0.9042865037918091, 0.9232165217399597, -0.6045728921890259, -0.4762580692768097, 0.07055123895406723, -0.18925048410892487, 1.7285945415496826, -1.3109345436096191, -0.07651932537555695, -1.2747080326080322, -0.38249096274375916, 1.1223558187484741, -0.43294480443000793, 1.1665104627609253, 1.1221827268600464, 0.6043115854263306, 0.23621295392513275, -1.5798896551132202, 0.8306674957275391, 1.3064388036727905, -1.4402453899383545, 1.3765137195587158, 0.7303767800331116, 1.0852091312408447, 0.6285818219184875, -0.7016434073448181, -1.1179993152618408, -0.3588836193084717, 0.9049011468887329, 0.15953078866004944, -1.9475284814834595, 0.37092021107673645, -1.0727479457855225, 0.7879387140274048, 1.0191679000854492, -0.10915099084377289, 0.956075131893158, -0.9484652280807495, -0.733289897441864, -0.14330853521823883, 2.826683521270752, 1.512513518333435, -1.1251102685928345, 0.47386959195137024, 1.5493592023849487, 0.543188750743866, -0.0676257386803627, -0.5893687009811401, -0.7585282921791077, -0.6287307143211365, 0.6340675950050354, -1.7724967002868652, 1.1606955528259277, 0.7717592120170593, -1.8755525350570679, 0.5658378005027771, -0.4470282793045044, 0.8693203330039978, 0.6908916234970093, -2.18815279006958, 0.22587785124778748, 0.3563046157360077, -1.5486202239990234, -0.3403826653957367, -0.8303804397583008, 1.783671498298645, -0.602136492729187, -0.3209247589111328, -0.5007091760635376, 0.20747150480747223, -0.2045375108718872, -1.2237036228179932, -1.0312913656234741, -0.4831792712211609, -1.6773663759231567, -0.38044777512550354, -0.995172917842865, 1.033852219581604, 0.7401590347290039, -3.358356475830078, -0.1456466168165207, -0.8070348501205444, -0.9165350198745728, -1.7085577249526978, 0.4481493830680847, 0.025469180196523666, -2.064458131790161, 0.18069402873516083, -0.7589549422264099, 0.717860758304596, -0.294558584690094, 0.8083502650260925, 0.6096851825714111, -1.579197883605957, 0.9028038382530212, 0.555634081363678, 1.3512200117111206, -0.5235131978988647, 1.4673242568969727, -0.4006500840187073, -0.4350493848323822, 0.7024211883544922, 1.4863686561584473, -0.6401503086090088, -0.9208300113677979, 2.1370816230773926, 1.7172538042068481, 0.041128430515527725, -2.0281546115875244, 0.007966416887938976, -0.31642934679985046, -0.45089852809906006, -0.7097839713096619, -1.371268630027771, 0.9037988781929016, -0.668427586555481, 1.5638316869735718, 0.4604361057281494, 0.9176647067070007, -0.13416258990764618, -1.2987470626831055, 0.1604403555393219, 1.3701447248458862, 0.9226532578468323, 0.6292062401771545, -0.17409832775592804, 1.6257975101470947, -1.3958348035812378, -0.542252242565155, -0.14111913740634918, 0.28441354632377625, 0.0835883766412735, -0.8719268441200256, -1.7447468042373657, 0.3080877363681793, 1.0771338939666748, -0.0053474269807338715, -0.2659386098384857, -0.6909063458442688, -0.12003704905509949, 0.7553306221961975, 0.8158321380615234, 2.486366033554077, -1.2057139873504639, 0.2939976751804352, -1.3748036623001099, 1.9050439596176147, -0.44527602195739746, -0.7732961773872375, 0.7408646941184998, -0.327178031206131, -0.355847030878067, -0.5617845058441162, 1.1407915353775024, -1.2925831079483032, -0.11061996221542358, -0.2974035441875458, -0.36135926842689514, 1.244538426399231, 0.7316019535064697, 0.190618634223938, 0.958558201789856, -0.12456288933753967, -0.22624020278453827, 1.0211955308914185, -0.310549259185791, 0.7105352878570557, 0.44250330328941345, 0.12163367122411728, -1.5036638975143433, 0.4852369725704193, -0.039015114307403564, -0.4715132415294647, -0.7743865847587585, -0.01373455673456192, 0.2082074135541916, 0.35443150997161865, 1.463734745979309, 1.380528211593628, 0.4327092170715332, -0.754964292049408, 0.0468880832195282, 0.3810522258281708, -0.9480198621749878, -1.9302642345428467, -0.6037406921386719, -0.4175397753715515, -1.4302109479904175, -2.4252941608428955, 1.3263689279556274, 0.23938269913196564, -1.5125802755355835, -0.001719965017400682, 1.5409958362579346, 0.538591206073761, 1.0373181104660034, 1.4267785549163818, 0.9417922496795654, -1.3759729862213135, 0.785347044467926, -0.5931710004806519, 0.35753506422042847, -0.2024359554052353, 0.6852032542228699, -0.308234840631485, 0.6197538375854492, 0.6195173859596252, 0.0563165657222271, -0.292477011680603, 0.2978071868419647, -0.3868272304534912, -0.32556822896003723, 0.519200325012207, 0.01560619380325079, -0.6974759697914124, -0.221368208527565, 0.6697013974189758, 1.3628133535385132, -0.6513296365737915, 1.1038233041763306, -0.7229870557785034, -0.8412370085716248, 0.15998083353042603, -0.1324257254600525, -0.2692986726760864, -0.4856739342212677, -0.05766227841377258, -0.3873824179172516, 0.3045176565647125, 0.5960844159126282, 0.47329288721084595, -0.9289163947105408, -0.40085020661354065, 0.027081694453954697, -0.04123982787132263, -0.5427564978599548, 0.45106491446495056, 2.231419563293457, -0.510299026966095, -0.06170100346207619, -2.4064431190490723, -0.7495928406715393, -0.1368144154548645, -0.24434854090213776, -1.2614576816558838, -1.133535623550415, 1.772993564605713, -1.002776026725769, -1.7129228115081787, -0.1098078042268753, -0.008674772456288338, -0.8611310124397278, 2.081512928009033, 0.37530311942100525, 0.6850876808166504, 1.0871379375457764, -1.1040637493133545, -0.21119922399520874, 0.0836079865694046, 2.1225194931030273, -0.5088139176368713, -0.08858486264944077, -0.5944700241088867, 0.9425588846206665, -0.1448240578174591, 1.3909639120101929, 1.0328104496002197, 1.0486938953399658, -0.5129730701446533, -0.14391274750232697, 0.4370169937610626, -1.0041797161102295, -1.2570345401763916, 0.20767365396022797, -0.18515285849571228, 2.172299385070801, 0.06952229887247086, -0.2020609825849533, -0.4560558795928955, 1.035326361656189, -0.3273525536060333, 0.29059985280036926, -0.393785297870636, -1.5290096998214722, -0.023242000490427017, 0.956726610660553, 0.14771676063537598, -0.23965121805667877, -1.657856822013855, 0.6862621307373047, -0.7750788927078247, -0.9138160943984985, 1.8202526569366455, 0.4326528012752533, -0.4248324930667877, -0.28303050994873047, -0.42029592394828796, 1.5068128108978271, -1.2704195976257324, -0.08765196055173874, 0.33320900797843933, -0.040630586445331573, -0.32105475664138794, -0.14485561847686768, 0.13652698695659637, 0.08293675631284714, 0.6461482048034668, -0.2707797884941101, 1.9524104595184326, 0.7978717684745789, -1.0765398740768433, -1.1617952585220337, 0.44647228717803955, -1.379939079284668, 0.9973568320274353, 0.9137706756591797, -0.023226359859108925, -2.1242499351501465, 0.15049223601818085, 2.4683663845062256, -0.6002007126808167, 0.8085826635360718, 0.4022952914237976, 0.15692386031150818, 0.48853012919425964, 1.0351665019989014, 0.11195168644189835, 0.6538468599319458, -0.009543807245790958, -1.5647708177566528, 0.3720662593841553, -1.1285877227783203, -0.7269497513771057, -0.6248769164085388, -0.6346948742866516, -2.239901065826416, 0.5379173755645752, 1.9449241161346436, -1.8813788890838623, 0.04605662822723389, -2.1912341117858887, -1.8484375476837158, -1.0592292547225952, -1.1405072212219238, -1.0472437143325806, -1.262532353401184, 1.333452820777893, 1.22675621509552, -0.5223037600517273, -1.0369263887405396, -0.2952953577041626, 0.5862935781478882, 0.8717765212059021, 0.16040319204330444, 0.703235387802124, 0.7706682682037354, -0.7450262308120728, 1.5706415176391602, 0.4065703749656677, 0.43091294169425964, 0.12168620526790619, 0.9077539443969727, 1.165928602218628, 0.09456251561641693, -1.7445628643035889, 0.07889071106910706, -0.546796977519989, -1.8798110485076904, -1.632320761680603, 1.8017622232437134, -0.19243255257606506, 0.0916154757142067, -0.7022181749343872, 0.9120693206787109, 4.19501256942749, -1.0126110315322876, 0.3072636127471924, 0.31846165657043457, -1.0197334289550781, -0.8745213150978088, -0.2758433222770691, 0.6392130255699158, -0.9779393672943115, 0.30847734212875366, -0.23576347529888153, 1.9882574081420898, -0.5096895694732666, 1.9031888246536255, 1.3124293088912964, -0.9132328033447266, -1.6354092359542847, 1.331299066543579, 0.7073971629142761, -1.098816156387329, 2.5336875915527344, -0.5666410326957703, 0.6795479655265808, -1.1206834316253662, -1.283732533454895, 0.9137422442436218, 2.1564736366271973, 1.1348634958267212, 0.6971839666366577, -0.10177796334028244, -1.9661917686462402, -0.4519205093383789, 1.0915350914001465, -0.04761224240064621, 0.23035241663455963, -1.605866551399231, -0.0803656354546547, -0.46707144379615784, 0.30098986625671387, 0.08644486963748932, 1.2523553371429443, -0.9406082630157471, 1.904946208000183, -1.1352453231811523, 1.435633659362793, -0.6663410067558289, -1.1663213968276978, 0.389149010181427, -1.6968971490859985, 0.4169634282588959, 1.1295149326324463, 0.9509645104408264, -1.1603189706802368, 0.35991358757019043, 0.3907913267612457, -0.6209520101547241, 0.9141801595687866, 0.48492488265037537, 2.2249910831451416, -0.8352208733558655, 0.30277693271636963, 0.8238649964332581, -0.6403762102127075, 0.6425280570983887, 0.21211448311805725, 1.687720537185669, -0.23338301479816437, 0.44077110290527344, 1.0721992254257202, -0.8712090849876404, -0.38143986463546753, -0.4832700490951538, 1.2994074821472168, 0.12802733480930328, -0.22715884447097778, -0.23769713938236237, -0.06145969405770302, -0.014734945259988308, 0.061294909566640854, 1.705733299255371, -0.3514387905597687, 0.3504117727279663, 0.3568996787071228, 0.16407068073749542, 0.0029665902256965637, 0.18853311240673065, 1.2563172578811646, -0.36831724643707275, -0.20312468707561493, 0.04102263227105141, -1.2846275568008423, -0.621232271194458, 1.4768849611282349, -1.1580170392990112, -0.6477636694908142, 0.5979332327842712, 1.1979702711105347, 0.26349666714668274, -0.7471375465393066, -0.758467972278595, 0.3338446021080017, -1.0889999866485596, -1.262976884841919, 0.48822498321533203, -0.4333541989326477, -1.9567917585372925, 0.5226935148239136, 0.9717113971710205, 0.8478204011917114, 2.056104898452759, 1.008120059967041, -1.9069342613220215, 1.4436100721359253, 0.19871021807193756, 1.5704303979873657, -2.4005649089813232, -2.4006588459014893, 0.6234431862831116, -1.1550136804580688, -1.6878865957260132, -1.6203855276107788, 0.29597562551498413, 2.6543736457824707, -0.27541136741638184, -1.188688039779663, 0.38836318254470825, 0.9104950428009033, 0.8250115513801575, 0.9622082710266113, -0.9141257405281067, 1.0522385835647583, -0.13065049052238464, -1.3190027475357056, 0.5116460919380188, -0.04063398391008377, 0.5588873624801636, -1.2976658344268799, 0.046460673213005066, 0.3860551714897156, 1.519045352935791, 0.06270977854728699, -0.8170082569122314, 0.4645761251449585, 0.30056098103523254, -0.20159897208213806, 0.43812671303749084, 0.14565248787403107, -1.4127931594848633, 1.304050087928772, -0.6228931546211243, -0.08486545085906982, 2.4736528396606445, -0.8188994526863098, 0.5797941088676453, 0.17757517099380493, -1.032721996307373, 0.3256101906299591, -2.4989945888519287, 0.19757403433322906, -0.7787479162216187, 0.825445294380188, 2.033679246902466, 2.1150057315826416, 1.599676251411438, -0.18632538616657257, -0.983272910118103, -0.624191403388977, -0.18896760046482086, 0.1583317667245865, 0.029383663088083267, -0.11594948917627335, -0.05170758068561554, 0.052611254155635834, -1.9349420070648193, -0.3094497621059418, 0.7998108863830566, -0.9096592664718628, 0.19230353832244873, -1.045395851135254, 0.393684446811676, 0.6680821776390076, 0.845434844493866, 1.523456335067749, -0.7273213863372803, -0.24935650825500488, 0.04510567709803581, -0.17137546837329865, 0.10833892971277237, 0.0862925574183464, 0.010370719246566296, -0.2854539752006531, 0.07921338826417923, -0.8576338887214661, -0.5982145071029663, -1.5858570337295532, 1.3222708702087402, 1.6259573698043823, -1.144955039024353, 1.196179986000061, -1.7911972999572754, 1.0777416229248047, -0.3269972801208496, -2.70334529876709, -0.392141193151474, 0.682414174079895, 1.641598105430603, 0.3350830674171448, -0.8495008945465088, 1.854623794555664, 0.09786194562911987, 0.6902884244918823, 0.5789375901222229, -0.7466843724250793, 0.9674460291862488, 0.6320878863334656, -0.24869820475578308, -0.4491004943847656, 0.6125166416168213, -0.9037880301475525, 0.09108560532331467, 0.784945011138916, -0.00014627582277171314, -0.44375893473625183, -0.2615758776664734, -1.3507391214370728, -0.9148958921432495, -1.4644616842269897, -1.5464253425598145, -1.3924918174743652, -1.1776641607284546, -1.1943104267120361, 0.2565533518791199, -0.38045620918273926, -0.7034271359443665, 0.4113312363624573, 0.15130452811717987, -0.2462998628616333, 0.21100713312625885, -0.34653544425964355, 1.5061479806900024, 0.8780009746551514, -1.002629041671753, -0.3888320028781891, 0.13878856599330902, 0.18543966114521027, 1.147800087928772, -0.17976970970630646, 0.18033502995967865, 0.9441472887992859, -2.1255369186401367, -1.047303318977356, 1.06149160861969, 0.6137020587921143, -1.3781001567840576, -0.7452248930931091, -0.8823365569114685, -0.7786924839019775, 0.8055638670921326, 1.9786854982376099, -1.0448650121688843, -1.3879364728927612, 1.4747012853622437, 1.2942118644714355, -1.2796815633773804, 2.080460786819458, 0.40056633949279785, 0.5638874769210815, 2.223555326461792, -1.0462608337402344, -0.856602132320404, 0.9244375824928284, 0.8654475808143616, 0.5275096893310547, 3.6244990825653076, 0.2548247277736664, 1.6320490837097168, 0.6190783381462097, -0.19966870546340942, 0.07639013975858688, 2.228644609451294, -0.12204111367464066, -0.10989227145910263, -1.294467568397522, -0.65411376953125, -0.7220151424407959, 0.549170732498169, -0.7843836545944214, -0.5625284910202026, -0.5074844360351562, -1.6769397258758545, -0.622555673122406, 0.6170315742492676, 0.9847593307495117, 0.4016702175140381, 0.5370194315910339, -1.2539610862731934, -0.41260430216789246, -0.6201086044311523, 0.17452284693717957, 0.8506278991699219, -1.5309460163116455, 0.11247946321964264, -1.8772876262664795, 1.8380475044250488, 0.4041614234447479, -0.6227211952209473, 0.7049511075019836, -0.32169464230537415, 0.30781373381614685, -0.06570926308631897, -1.3482290506362915, -0.30663836002349854, 0.5975023508071899, 0.7019367814064026, 1.912078857421875, 0.5666046142578125, -1.7772516012191772, 0.5257574319839478, -0.34104952216148376, 0.6825229525566101, 0.6208176612854004, -0.6272026300430298, -1.8645517826080322, -1.2649176120758057, 1.5330883264541626, 0.04149051755666733, -0.9247634410858154, -0.13971547782421112, -0.21797871589660645, 1.9995862245559692, 0.3499079942703247, -0.6551769971847534, -0.6905252933502197, -1.0661280155181885, 0.27809083461761475, 1.4162746667861938, 0.3219946026802063, -0.30281367897987366, -0.5403279066085815, -0.1764296144247055, -0.33628058433532715, -1.1870040893554688, -1.3170033693313599, 1.2492586374282837, -0.6192932724952698, 0.4886932075023651, 2.0563483238220215, -0.731822669506073, -0.5238286256790161, -0.6174394488334656, -1.75933039188385, 2.6848418712615967, 0.9695762991905212, 0.20365335047245026, 0.7951281666755676, 1.2661457061767578, -1.3246439695358276, -0.9784440994262695, 0.3741920590400696, -0.07648077607154846, -0.9110430479049683, -0.2260817587375641, -2.0853726863861084, -0.522071361541748, 1.2874456644058228, 1.1437321901321411, 0.9235387444496155, 0.22839950025081635, -1.5016018152236938, -1.8401528596878052, 0.32937899231910706, 1.008319616317749, 1.0496292114257812, 1.192674994468689, -0.6321888566017151, -0.6667653322219849, 0.036100033670663834, -1.4337871074676514, -1.3123562335968018, 0.43033069372177124, -0.7189820408821106, 2.447847366333008, -1.1405776739120483, -1.2673276662826538, 0.2222549021244049, -1.1897978782653809, 1.0756187438964844, -0.07335280627012253, -0.9477443099021912, 0.020751696079969406, -2.7341604232788086, -0.6824325323104858, -0.7088122367858887, 1.1369317770004272, 0.22283293306827545, 1.2635749578475952, -1.5642082691192627, 1.4092743396759033, -0.2479587197303772, -0.22065693140029907, 1.3377918004989624, 1.289801001548767, 1.304884910583496, -2.691291332244873, -0.23248329758644104, 1.899077296257019, -0.07718997448682785, -1.324897050857544, -0.5903505086898804, -1.0016387701034546, -0.7669205665588379, 0.7338331341743469, 0.43636763095855713, 1.4068468809127808, 1.828963279724121, -1.258832573890686, 0.5989872217178345, 1.2219158411026, -2.9742493629455566, 0.10984641313552856, 0.5939214825630188, 0.891373336315155, 1.9280407428741455, -0.1524394303560257, 0.02892916463315487, -0.062349628657102585, 0.936194121837616, 0.7437745928764343, -1.35159432888031, -0.46202993392944336, 1.1260720491409302, -1.4170068502426147, 2.257992744445801, 0.5462249517440796, 0.8450666666030884, -0.6708000898361206, -0.20739351212978363, 0.46061569452285767, -0.02948801964521408, 0.08176422864198685, -0.30523785948753357, 0.00651167705655098, -1.2022418975830078, 0.7369298338890076, 0.14509883522987366, -0.38135823607444763, 0.019213680177927017, 0.05127879977226257, -1.0785642862319946, -0.03140229359269142, 0.7422443628311157, -0.3689110577106476, 0.269275963306427, 0.4519045650959015, -0.7756619453430176, 0.6461630463600159, 0.7656537294387817, 1.3234491348266602, -0.28423482179641724, -0.2584753632545471, -0.9244476556777954, 0.9312940239906311, 0.49010589718818665, 0.3022363781929016, 0.6123825311660767, 0.9207110404968262, 1.3524913787841797, -0.06895279884338379, 1.5732837915420532, -0.1624123454093933, 2.114866018295288, 0.08459743857383728, 0.5037065744400024, 0.7079744338989258, -0.5799058079719543, -1.6736630201339722, 1.5113548040390015, 2.244539737701416, 0.8929606676101685, 1.4863426685333252, 0.515536904335022, -0.6334744095802307, -0.6145778298377991, 0.7670387625694275, 1.2312912940979004, -0.24672549962997437, -0.7256496548652649, -0.3174252212047577, 0.04436951130628586, -2.168816328048706, 1.2048687934875488, -1.4850215911865234, -0.9720684885978699, 0.29648950695991516, -0.27875587344169617, -1.5353858470916748, -1.3693004846572876, -1.2337851524353027, 2.0259087085723877, -1.30937659740448, -1.2074476480484009, 1.2972862720489502, 0.5613507032394409, 0.3466911315917969, -1.6443406343460083, -1.8493295907974243, -0.4842212498188019, -0.30926087498664856, 1.0318424701690674, 1.352944254875183, 0.2328292429447174, 0.20494484901428223, 1.983172059059143, -0.5761064291000366, -1.3948262929916382, -1.241182565689087, 0.8094090223312378, -0.5589438080787659, -0.4516885578632355, -1.4331474304199219, 1.376183032989502, -1.4185630083084106, -0.04682274907827377, 0.2858804166316986, 0.6221352219581604, -0.7246930599212646, -0.3685831129550934, -0.3034747838973999, 0.07764244824647903, -0.5462101697921753, 1.2629152536392212, 2.3458943367004395, 0.07894858717918396, 0.35539600253105164, -0.0760548934340477, 1.7559932470321655, -0.9363076090812683, 0.8996353149414062, 0.22749482095241547, 0.7406378388404846, -0.6980745196342468, -0.5066519379615784, -0.640367329120636, -0.4768087863922119, -0.10151268541812897, -2.1288769245147705, 0.2750866115093231, 0.4822261929512024, 0.6993663907051086, -0.8039858937263489, 0.5221391916275024, -0.5353742837905884, 0.2332606166601181, 0.592250406742096, 0.8866063952445984, 1.943674921989441, -0.5768500566482544, -0.7713380455970764, -1.678026795387268, 0.021337157115340233, 4.548691749572754, -1.4498006105422974, -0.14726804196834564, 0.5733386874198914, -0.5223255157470703, 1.5892950296401978, -0.6297679543495178, 0.14914007484912872, -1.0810859203338623, -1.2288202047348022, 0.6957874894142151, 0.7471024394035339, -0.7147406935691833, 0.5185121893882751, 0.16470015048980713, 0.7374171614646912, 0.5356206893920898, -0.20259033143520355, 0.99094557762146, -1.927757740020752, -1.7294788360595703, -0.3756659924983978, 1.3272308111190796, 1.444143295288086, -0.7286593317985535, -0.11888296157121658, -1.1937285661697388, -0.08192549645900726, -1.712843894958496, -0.14965999126434326, 0.282446026802063, -1.826200008392334, -1.033341646194458, 0.8646987080574036, 0.2835521399974823, -1.5054426193237305, -0.19021739065647125, 0.7364504933357239, -1.3893377780914307, 0.5039790868759155, 0.11042506247758865, 0.04575304314494133, 1.143210530281067, -0.8625910878181458, 0.3921523988246918, 1.3763421773910522, -0.12781381607055664, -0.4789372980594635, -0.48471176624298096, 0.28392311930656433, -0.15516771376132965, -0.7741401195526123, -0.6880537271499634, 1.0002734661102295, 0.2832995057106018, 1.585991621017456, 1.6043150424957275, 1.3901796340942383, -0.41998210549354553, 0.4822763204574585, 1.0658535957336426, -0.12783168256282806, -0.2069416046142578, -0.5974158644676208, 1.3452082872390747, -0.08518574386835098, 1.101114273071289, -0.5608804225921631, -1.4925339221954346, 0.3616604506969452, -1.3871968984603882, -1.017686128616333, -2.180309772491455, -1.6527246236801147, -0.20738551020622253, -0.5019740462303162, -0.621566653251648, 0.40144094824790955, 0.01498655416071415, -0.2547874450683594, -0.3878375291824341, 1.7023134231567383, 0.8729652762413025, -1.0049077272415161, -0.6450740694999695, -1.3834911584854126, -0.26722705364227295, -0.5221177935600281, -0.10937381535768509, 1.0642082691192627, -0.04445718601346016, 0.64041668176651, -0.548157274723053, -0.2178921401500702, -0.39559000730514526, 1.2835261821746826, -0.9095717072486877, 2.0089046955108643, 0.33883678913116455, -0.9408420920372009, -0.8632524013519287, -0.25111526250839233, -0.4046962857246399, -0.04095457121729851, -0.23849929869174957, 0.7044427990913391, -1.9002797603607178, -0.5642310380935669, 0.4864045977592468, 0.030285928398370743, -1.2441116571426392, -0.10568107664585114], [-1.3198038339614868, -1.0801934003829956, 0.520887017250061, -0.9885274171829224, 0.456158846616745, 0.3007013201713562, 0.7045171856880188, -0.47785159945487976, -0.8821553587913513, 0.017705563455820084, -0.5829738974571228, -1.068034291267395, -1.743613839149475, -1.4340934753417969, 1.040889859199524, 1.2368860244750977, -0.6510524153709412, 1.1442749500274658, 0.14558883011341095, -0.21257363259792328, -0.9149149656295776, 0.18255014717578888, -0.026781348511576653, 0.3619360029697418, -0.12542083859443665, -0.7569092512130737, -0.5573325753211975, -1.0398226976394653, 0.5766825079917908, 1.913773536682129, -1.027086615562439, 1.1127979755401611, -0.4004686176776886, -0.3905056416988373, -0.9660789370536804, 0.31192219257354736, 0.8041042685508728, 0.5115893483161926, -0.511910080909729, 0.6562591195106506, -1.2635836601257324, -1.2857701778411865, -0.43290263414382935, -0.06421666592359543, -0.8181543946266174, 0.6037119626998901, -0.2217458337545395, -0.3858168423175812, -0.1755988448858261, -0.10648924112319946, -1.4740691184997559, 0.24984073638916016, 2.4152660369873047, 0.46176576614379883, 0.1570499688386917, 0.4345554709434509, -0.1211550235748291, 0.028148217126727104, 0.9320225715637207, -0.4965646266937256, 2.5377073287963867, 2.9150829315185547, 0.15883050858974457, 0.7835967540740967, -1.5117233991622925, 0.6911125183105469, -1.9057801961898804, -1.3344025611877441, 0.10865043848752975, -0.39126431941986084, -0.34594306349754333, 0.15409666299819946, 0.29078903794288635, 1.1640346050262451, 3.332212209701538, -0.388969361782074, -0.23166270554065704, 0.13602648675441742, 0.5573753714561462, 1.0573140382766724, 1.5031908750534058, 0.49418407678604126, 1.6461530923843384, 0.6749188303947449, -0.4133967459201813, -0.842022180557251, -1.8220114707946777, -1.1353987455368042, -2.350883960723877, 1.4504073858261108, 0.12213815748691559, 0.01629600301384926, -0.23233138024806976, -0.3995235562324524, -2.6186025142669678, 2.062760353088379, 2.285264253616333, -1.6243205070495605, -0.9884687066078186, -0.49535492062568665, 1.0543498992919922, 1.106703519821167, 0.5804293155670166, 0.6036496162414551, 0.6374862194061279, -0.9813804626464844, 0.7880672216415405, 0.5555595755577087, 0.9802528023719788, 0.038650162518024445, -0.6159611344337463, 1.1271493434906006, 1.6089979410171509, 0.7571010589599609, 1.4722260236740112, -0.32298362255096436, -0.7304290533065796, -1.2249435186386108, 0.29018551111221313, -1.099374532699585, 0.5681141018867493, 0.921988308429718, -0.6861546635627747, -0.6242345571517944, 0.625468909740448, 1.580926775932312, 0.9545836448669434, -0.34303948283195496, 0.024145174771547318, 2.294348955154419, -2.0826971530914307, 1.8586347103118896, -0.15093982219696045, -0.5955601930618286, -1.289629578590393, 0.838346540927887, 1.2901549339294434, -0.6241579651832581, 0.14078760147094727, -0.35385245084762573, 1.6418664455413818, 1.619099736213684, -1.367639422416687, -0.49825379252433777, -0.133693665266037, -0.252240926027298, 1.0796453952789307, 0.2740316390991211, -1.0900341272354126, 0.18416699767112732, 0.1241021528840065, -0.6837707757949829, 0.9165416955947876, -0.40332403779029846, -0.40225905179977417, 1.8542678356170654, -0.16909481585025787, -1.748194694519043, 0.09040692448616028, 1.0617929697036743, -0.015389676205813885, -0.1959846466779709, -1.2743843793869019, 0.08300391584634781, -1.2956817150115967, -0.9542666673660278, 0.2735622525215149, 0.39127984642982483, 0.20248225331306458, -0.8635625839233398, 0.7727146148681641, 1.152587652206421, -0.19444158673286438, -1.705386996269226, -0.22896309196949005, -1.380808711051941, 0.39815422892570496, 1.9711313247680664, -0.15995699167251587, -2.1347012519836426, 0.7759464383125305, -0.127200186252594, 1.077459454536438, -1.2281644344329834, 1.126158595085144, -0.9127405881881714, 0.028361031785607338, -1.072971224784851, 0.36021459102630615, 0.9334191679954529, 0.07332377135753632, 1.958817720413208, 0.8418428897857666, -0.11865226179361343, -0.9104098677635193, -0.7261269092559814, -0.346103310585022, 1.6492851972579956, 1.0421565771102905, -1.0117470026016235, 0.7205098867416382, 0.34940221905708313, -1.680848240852356, -0.6784950494766235, 1.637420415878296, 0.4411573112010956, 0.02007286623120308, -0.468593031167984, -0.42006659507751465, -0.573743999004364, 1.1922225952148438, 1.1105629205703735, -0.4530937373638153, -1.4101970195770264, -0.12939900159835815, -0.5753431916236877, 0.6163331866264343, -0.06600993126630783, 0.050331391394138336, -0.08809860050678253, -1.0928269624710083, 0.931524395942688, -1.4660534858703613, -0.685808539390564, -0.5633507966995239, -1.3768144845962524, -1.6312698125839233, 0.6964046359062195, 2.7849953174591064, 0.9723803997039795, -0.5098908543586731, -1.0065150260925293, -0.18972738087177277, 1.449442744255066, -0.3322666585445404, 1.2855395078659058, 1.5492808818817139, -0.6935751438140869, -0.7636454701423645, 1.535796046257019, 0.5826114416122437, -0.6327294111251831, -0.040652088820934296, -0.7001081705093384, 0.15508732199668884, 0.24920624494552612, -0.27427229285240173, -1.6820666790008545, -0.26758161187171936, 0.6238711476325989, -0.30039182305336, -0.9534629583358765, -0.7677763104438782, 0.8388611674308777, -0.9101921916007996, -0.7990970611572266, -0.14928382635116577, 0.18229033052921295, 0.37430015206336975, 0.13548363745212555, 1.2313817739486694, 0.6709232330322266, -1.6389306783676147, -0.24453233182430267, -0.15158505737781525, 0.8559487462043762, 0.5103841423988342, -1.5697966814041138, -0.8171544075012207, 0.38795337080955505, -0.4589272737503052, 1.3814181089401245, 0.8211135268211365, 1.393681287765503, -0.12084535509347916, 1.7152012586593628, 1.4774670600891113, -0.23070593178272247, 0.5138767957687378, 1.776231288909912, 0.364728182554245, 0.15151670575141907, 0.5180633068084717, 0.6406221389770508, -0.48286813497543335, -1.247188925743103, -0.6852978467941284, -0.7382851243019104, 1.2016462087631226, -1.952457308769226, -2.401337146759033, -1.15263831615448, 1.1942044496536255, -0.5727875828742981, -0.6865311861038208, -1.3196018934249878, -0.7555550336837769, -0.9799507856369019, 0.37220603227615356, -0.4352051317691803, -0.5329628586769104, 0.7391232848167419, 1.7898969650268555, -1.3921451568603516, -1.1501119136810303, 0.47528931498527527, 0.26055553555488586, 0.20750528573989868, -0.18130506575107574, 0.12002153694629669, 0.5966653227806091, -1.0935572385787964, 0.3003706634044647, -0.6249344348907471, -0.3048350214958191, -0.36463290452957153, -0.31531718373298645, 0.37537190318107605, 0.35071006417274475, -0.836879312992096, -1.409580111503601, 0.3963685929775238, 0.4381612241268158, 0.09444296360015869, -1.2519792318344116, 0.713468611240387, -0.1370088905096054, -0.5540123581886292, 1.6416321992874146, -0.41138410568237305, 1.240096092224121, -0.464160680770874, -0.7696965336799622, 1.7312028408050537, 0.4423830211162567, -1.685875654220581, -0.3062867820262909, -0.4540821313858032, -0.30445584654808044, 0.9188799858093262, -1.1701408624649048, -0.29217761754989624, 0.6334540247917175, -0.15161585807800293, -0.6763995885848999, -0.03867780417203903, 2.094730854034424, 0.7091720700263977, -0.20343346893787384, 0.9034276604652405, -1.1329683065414429, 0.5277955532073975, 0.131052628159523, 1.0617326498031616, 1.828580617904663, -0.11442331224679947, -0.07461916655302048, -0.49595656991004944, 0.5365297794342041, 1.1803077459335327, 0.5886984467506409, 1.3804113864898682, -0.33535370230674744, -1.9680663347244263, 1.5716171264648438, 0.29486384987831116, -0.4453420341014862, -2.599179983139038, 0.9695631265640259, 0.4057500660419464, 0.2736867666244507, -0.25816866755485535, -0.20863467454910278, 0.5420226454734802, 0.7898687124252319, 1.371134638786316, 0.7046955823898315, 1.2517751455307007, 0.8349277973175049, 1.007932424545288, 0.5659152269363403, 0.7435640692710876, 1.6022679805755615, 1.149144172668457, -0.2753179967403412, 0.01683095097541809, -0.22242680191993713, 1.4073299169540405, 0.40286147594451904, -0.43768319487571716, -0.9675119519233704, -1.174473762512207, 0.33809348940849304, -0.2227754443883896, 0.13867978751659393, -1.5316261053085327, -0.03796372190117836, 1.0522587299346924, -1.106877088546753, -0.12831427156925201, -0.20571394264698029, -0.24143700301647186, -0.7116231322288513, -0.6753615736961365, 0.9427968859672546, -2.1892752647399902, 0.16183142364025116, -0.4911849796772003, -0.7211899757385254, -0.26903772354125977, -0.9014972448348999, 1.5306060314178467, 0.3934038281440735, 0.9112600684165955, -1.1971358060836792, -1.38190495967865, -0.7914087176322937, 1.5005708932876587, 1.6610422134399414, -0.0557217113673687, 0.3062736690044403, -1.7744126319885254, 0.24366432428359985, -0.8103846311569214, -0.5600048899650574, -1.1926815509796143, -0.2543191909790039, -1.368062973022461, -0.11538959294557571, 0.5850138664245605, -1.2759935855865479, 0.19581304490566254, -0.04942562058568001, -0.628556489944458, 0.6380105018615723, -0.3699376583099365, -0.49682334065437317, 0.6938318014144897, 0.1781928986310959, -1.7509348392486572, 1.143877387046814, -1.968723177909851, -0.06333588063716888, 1.4877640008926392, 1.6283886432647705, -0.6977618336677551, -0.10050127655267715, 0.8097807168960571, 0.7072128653526306, 0.29621621966362, 1.5917025804519653, -1.2152577638626099, -0.6803773045539856, -1.5176562070846558, -0.9793397188186646, 0.20413877069950104, 1.9137530326843262, 1.4312244653701782, -0.2183547168970108, 0.33141040802001953, 1.3285112380981445, -0.6675620079040527, -0.2892065942287445, 2.4239814281463623, 1.5235573053359985, 1.3414729833602905, -1.0997856855392456, 0.8504669070243835, 1.702907681465149, 1.117750644683838, -1.6359777450561523, 0.08431260287761688, -1.614180326461792, 0.8356921672821045, -1.0040371417999268, -0.7828120589256287, 0.057172685861587524, -0.46003150939941406, 0.8083194494247437, -0.8249549269676208, -0.4876244068145752, 0.4038543403148651, -0.17411577701568604, -0.5919582843780518, -0.11438142508268356, 1.0060042142868042, 0.48836857080459595, 0.6377438902854919, -1.7180640697479248, 0.6629403829574585, 1.2114272117614746, 0.33446070551872253, -1.6956641674041748, -0.921044647693634, -0.9228844046592712, -0.2986222803592682, -0.24179384112358093, -0.863893985748291, 0.2248264104127884, 0.8273341655731201, -2.043940782546997, 0.870276927947998, -0.10970093309879303, -1.436815857887268, -1.9933793544769287, -1.0112826824188232, -0.08730830997228622, -0.028156926855444908, -1.9883651733398438, -3.3253636360168457, 0.2585150897502899, -0.9355475902557373, -0.14366818964481354, 0.29274794459342957, 0.13082146644592285, 0.20960381627082825, -0.9438438415527344, 1.0481425523757935, 0.25743433833122253, 0.3392603099346161, 1.5300135612487793, -1.1421171426773071, -1.129654884338379, 1.317582368850708, -1.331860899925232, 0.8916054964065552, -0.0810619369149208, -0.7040517330169678, 0.9097157716751099, 0.13945452868938446, 0.0021569118835031986, -1.0383484363555908, 0.842538058757782, 0.2670435905456543, -0.44133996963500977, 0.7693443298339844, -0.7134570479393005, -0.45582830905914307, -1.4958404302597046, -0.1498212218284607, -0.013774675317108631, -0.17566418647766113, -1.0062857866287231, 0.3567858040332794, 1.0988930463790894, -2.4263505935668945, -1.1450453996658325, -0.10301823914051056, 0.213938370347023, -0.41073891520500183, 1.70823335647583, 0.27401745319366455, -0.35168036818504333, -0.13413006067276, 1.7094413042068481, -0.5754261016845703, 1.9950443506240845, 0.26449304819107056, 0.3865286409854889, 0.7004938125610352, -0.6977195143699646, -1.2254912853240967, 0.45781049132347107, 0.9284148812294006, -0.15770141780376434, 0.7376514077186584, 1.436947226524353, 0.3763827085494995, -0.7197492122650146, -1.2711988687515259, 0.08372088521718979, -0.15889407694339752, -2.3035073280334473, -0.40446826815605164, -0.9786273837089539, -1.4268767833709717, 0.9155545234680176, -0.19162151217460632, 1.4003759622573853, -0.4914718568325043, -1.1812549829483032, -0.7861487865447998, 0.5222721099853516, 0.5987433195114136, 0.4335854649543762, -0.10016567260026932, -1.1790770292282104, -1.5242187976837158, 0.6151795387268066, -1.5886589288711548, 1.5348199605941772, -0.739753782749176, 1.101209044456482, -0.7560285329818726, -1.7696863412857056, 0.5027487277984619, -0.541981041431427, -1.0039455890655518, 0.7704243063926697, -0.20943880081176758, -1.2026516199111938, -0.17709968984127045, -1.8434120416641235, 0.15423114597797394, -2.2542128562927246, -0.5670207738876343, -0.7337621450424194, -0.3297627866268158, -1.5047478675842285, -1.7301932573318481, 0.9925300478935242, -2.1573712825775146, -1.073002576828003, 0.11051701754331589, -0.4585711359977722, 0.35212671756744385, -0.4844651520252228, -2.011500835418701, -0.3792356550693512, 1.8511064052581787, -1.461859107017517, -0.4255334436893463, -0.7737263441085815, 1.2522392272949219, -0.3334576189517975, 0.8296409845352173, -0.9983299374580383, -1.33692467212677, -0.3173595666885376, -0.6588753461837769, 0.9086866974830627, -0.4093424081802368, 1.0226619243621826, -0.012529895640909672, -1.1856626272201538, 0.18450124561786652, -0.22837947309017181, 0.38081905245780945, -1.5883255004882812, -1.0526444911956787, -0.4141794741153717, 0.9616206884384155, 0.6343116164207458, -0.14188985526561737, 0.21034902334213257, -1.5183358192443848, 0.21316878497600555, 0.6616748571395874, -0.3829140365123749, 0.08777016401290894, 0.1809353083372116, 1.1501973867416382, 1.1332875490188599, -0.769584059715271, 0.4841632544994354, 0.5210939645767212, 1.0543028116226196, -0.7884387969970703, -0.0057288422249257565, -2.2848668098449707, 0.4813339412212372, 1.178138017654419, 0.37084442377090454, -0.7667995095252991, -1.0540788173675537, -0.4981602132320404, 0.22347605228424072, 0.7809247970581055, 0.5768204927444458, -2.101426839828491, 0.6187266111373901, 1.5426350831985474, -0.36593690514564514, 0.6857719421386719, 1.616044282913208, -0.5111793875694275, 0.3959306478500366, 0.4300978183746338, -0.14771710336208344, -1.385949730873108, -0.22526362538337708, 0.1040138304233551, -1.0825985670089722, -1.7492985725402832, -1.7635942697525024, 0.3600117564201355, 1.4575012922286987, -0.3658616244792938, -0.018091095611453056, 1.0716875791549683, 0.10389900207519531, -0.5962433218955994, -0.800373911857605, 1.3735852241516113, -1.7467395067214966, -0.9515009522438049, 1.2658509016036987, -1.074881911277771, 1.2499743700027466, 1.8795491456985474, 2.522376537322998, 0.9787543416023254, 0.03460600972175598, 0.45742687582969666, 1.7253249883651733, -1.2649587392807007, -0.35077351331710815, -2.356524705886841, 0.9117571115493774, 0.3032280206680298, 0.45401084423065186, -1.2617974281311035, -0.5239329934120178, -0.6029264330863953, -0.9858730435371399, -0.22268490493297577, 0.754584550857544, 1.6953707933425903, -0.0896305963397026, -0.6896002292633057, 1.111427903175354, -1.1815401315689087, 1.3167405128479004, -0.32914039492607117, 2.9412682056427, 0.6346010565757751, 0.4485360085964203, -0.6746974587440491, 0.0866062268614769, -1.2024555206298828, 1.2654531002044678, 0.8694161176681519, 0.928604006767273, 0.5399635434150696, -2.076657772064209, 1.8265085220336914, 0.712818443775177, -0.07019960880279541, -1.5884584188461304, -0.6504492163658142, -0.4967810809612274, -1.2608039379119873, -0.3291761577129364, -0.36717647314071655, 0.2135733664035797, -0.9432059526443481, 1.7307995557785034, -0.7219293117523193, -0.7318862080574036, -2.6601641178131104, 1.2007344961166382, -0.23747988045215607, 0.8731604218482971, -1.3338149785995483, 0.7759065628051758, 0.9303691983222961, -0.41858047246932983, -0.02968822792172432, -1.8583775758743286, 0.43115925788879395, -1.0462620258331299, 0.1716836392879486, -2.1624789237976074, -1.397152304649353, -0.15529392659664154, 0.7931824326515198, 1.2080920934677124, 0.7644863128662109, 0.05639960616827011, 0.8258626461029053, 0.14630192518234253, 1.7497440576553345, -1.8824043273925781, 1.039258360862732, 1.550349235534668, 0.4487079977989197, 0.8397411704063416, -0.26104024052619934, 0.5095084309577942, -1.8150259256362915, -0.7441015839576721, -0.30678790807724, 0.12470047920942307, -0.9124847054481506, 0.01001607533544302, 0.5842123627662659, 1.7980438470840454, -0.9894216656684875, -1.357290506362915, -1.1502690315246582, 0.18963098526000977, -0.3563411831855774, 0.03545308858156204, -0.6162242293357849, -0.4570242166519165, 1.1781452894210815, -0.13302890956401825, -0.9034947752952576, 0.7823109030723572, 1.3728008270263672, -2.089263916015625, -0.3947688639163971, -0.8520339727401733, 0.7181016802787781, -0.1920323371887207, 0.2719317376613617, 0.9394133687019348, -1.6864629983901978, 0.5497860908508301, 1.5242223739624023, 0.25641828775405884, 0.9402216076850891, 1.0329495668411255, -0.8725820779800415, 0.32260793447494507, -0.08233626186847687, -0.4040418267250061, 0.24855254590511322, 0.4701387882232666, 1.2610952854156494, 1.8474153280258179, 0.6646361947059631, -1.205565333366394, -0.2008991688489914, -0.8187083005905151, -0.7104066014289856, 0.091720350086689, 1.818416714668274, 0.22906886041164398, 0.5385088920593262, -1.7532219886779785, -2.5526669025421143, 0.44459566473960876, -1.358945369720459, -1.1735750436782837, -1.7029011249542236, 0.8576006889343262, 0.30000239610671997, 0.2116750329732895, 1.2576849460601807, 0.08620376884937286, -0.45739904046058655, 0.5846457481384277, -0.4357910752296448, 0.8126789927482605, 1.1726714372634888, -0.3459095358848572, -1.270289421081543, 0.43892988562583923, -1.0260608196258545, 1.1734009981155396, -0.1786302626132965, -1.354255199432373, 1.0997865200042725, 0.8393560647964478, -0.4770185649394989, -0.305228590965271, -0.37207505106925964, -0.7563852667808533, -0.8705888390541077, 0.7424658536911011, -0.7701552510261536, 0.07081245630979538, 0.3223780393600464, -1.0504088401794434, 0.24827276170253754, -0.11793433129787445, 1.625640869140625, 0.9386122822761536, -0.5374664068222046, 1.2738122940063477, -1.1981228590011597, 0.0630970224738121, 0.22157667577266693, 2.014875888824463, -0.3186771273612976, 0.4834933876991272, -0.6713746786117554, -0.4902507960796356, -1.6235240697860718, -0.025119828060269356, 0.14372511208057404, -0.1295097917318344, 1.0469166040420532, -0.007550315000116825, 0.7729207873344421, 0.8244918584823608, 0.9735554456710815, -1.2667287588119507, 0.019758805632591248, -1.6276023387908936, -0.9023052453994751, -0.1633194386959076, 2.1571247577667236, 0.5861510038375854, -0.47574034333229065, 1.5550774335861206, 0.4405598044395447, 0.8800796270370483, -0.3222390413284302, 1.1527270078659058, -0.12918511033058167, -0.10463716089725494, 0.010766498744487762, 2.0029940605163574, 0.3445069193840027, -0.10818523913621902, 0.004036341793835163, 0.17467617988586426, -1.0119178295135498, -0.8335458040237427, -0.7310110926628113, 1.3795595169067383, -0.5324731469154358, -1.5130302906036377, 1.1005216836929321, -1.0282036066055298, 0.7383449077606201, 0.28924936056137085, -0.879166841506958, 0.737642228603363, 0.686000406742096, -0.14222019910812378, 0.30099403858184814, -0.970356285572052, 0.6580781936645508, -0.3696150779724121, 1.5497558116912842, -0.5241289138793945, 2.4809041023254395, 0.7096266150474548, -0.46712246537208557, 0.4653879404067993, -0.2134193629026413, -0.32759857177734375, -0.8561045527458191, 0.8033358454704285, 0.32219675183296204, 1.2397704124450684, 1.2072571516036987, -0.2161116749048233, 1.3698689937591553, 0.6069448590278625, 1.0082756280899048, -0.4898446500301361, 1.303410530090332, 0.08192668855190277, 1.5918145179748535, -0.6985774040222168, -0.29517972469329834, 0.21406257152557373, -0.10863825678825378, -0.41761958599090576, 0.4169265031814575, -0.4884045124053955, -1.121405839920044, -1.3785510063171387, 0.48126134276390076, 0.39487946033477783, -0.7516260147094727, -0.32922154664993286, -0.08617252856492996, 0.33881816267967224, 0.2710763216018677, 1.7557563781738281, 0.058185916393995285, 0.03729386255145073, 1.2517565488815308, -0.218437060713768, -0.6162407994270325, 0.307712197303772, 3.581935405731201, -0.20686529576778412, 0.2812666594982147, -0.15241992473602295, 0.13818413019180298, 0.29646700620651245, -1.3875815868377686, 1.0131447315216064, -1.0672019720077515, 0.0927288755774498, -1.6069815158843994, 1.094814658164978, -0.4372768998146057, 1.56657075881958, -0.7967084646224976, -0.32093238830566406, -1.0477961301803589, 0.6418744921684265, 0.5608188509941101, 0.5796710848808289, -1.1345086097717285, 0.252971887588501, -1.1422317028045654, 0.22156791388988495, -0.29514339566230774, -0.7063639760017395, -0.3085523545742035, 0.7988589406013489, -0.6792280673980713, 0.42734017968177795, 0.4163043797016144, -0.6082590222358704, -1.3348455429077148, -0.15093202888965607, -0.4895201623439789, 1.268941044807434, 0.3361586034297943, 0.7012627124786377, -0.7370734810829163, 1.4047545194625854, 1.9862004518508911, 0.3578486442565918, 0.4323403537273407, -0.32818296551704407, 0.11778858304023743, -0.7051019072532654, -0.9954491257667542, -1.3606840372085571, -0.9069509506225586, 1.383981704711914, -0.2842991352081299, -1.334700107574463, 1.1611924171447754, 2.40069317817688, -0.3657601475715637, -0.9422535300254822, 0.7746689915657043, 1.4474339485168457, -0.024340549483895302, -0.13932165503501892, -1.4810993671417236, -0.37865322828292847, -0.7364431023597717, 0.1405891627073288, -1.3781065940856934, -1.9951828718185425, -0.6348363757133484, 0.8157173991203308, -0.6914303302764893, 1.829690933227539, -0.691573977470398, 1.7227180004119873, -1.107961654663086, 0.2312363237142563, -1.1878081560134888, -0.4621134400367737, -0.3684311807155609, -0.5874764919281006, 1.1903049945831299, -0.0419265553355217, 0.47990062832832336, 1.1684160232543945, 2.022547960281372, -1.57355797290802, 2.5491371154785156, 0.8165760636329651, 0.33816441893577576, 0.8256107568740845, -2.106252670288086, 0.902916431427002, 1.7062031030654907, 1.4821388721466064, -0.8345645666122437, 1.1791610717773438, -1.2553410530090332, -0.7441872954368591, -0.04511215537786484, -1.4672931432724, -2.231259346008301, 0.47850924730300903, -0.02905879355967045, -0.13745348155498505, 0.5008718967437744, -0.2905641496181488, 0.12610991299152374, -1.0457289218902588, -0.325141966342926, 1.728340983390808, 0.7430841326713562, 1.333325982093811, -1.2838321924209595, 0.07702337205410004, -0.14868295192718506, 0.23465298116207123, -1.8842610120773315, -1.3386915922164917, 0.24175839126110077, 2.0960187911987305, -0.3583340644836426, 0.5084229707717896, 0.04404238611459732, -0.3127252757549286, 0.8361331224441528, 0.31863224506378174, 0.7589431405067444, -0.007109584752470255, 0.783841073513031, -0.5707780718803406, -0.1672361046075821, -1.1230053901672363, -0.01753014326095581, -0.4639120101928711, -0.18697749078273773, 1.3683316707611084, 0.09572294354438782, -0.40007615089416504, 0.39650923013687134, 0.7533280253410339, 0.22346128523349762, 1.2539517879486084, -0.4897749722003937, -0.23147325217723846, -0.7223934531211853, -0.0688590258359909, 1.170520544052124, -0.1760663539171219, 2.5455830097198486, 0.2626560628414154, -1.0824265480041504, 1.3242071866989136, 0.016143465414643288, -0.47202351689338684, -0.4498315453529358, -2.2645468711853027, -0.22018851339817047, 0.9229893088340759, 0.3795642554759979, -0.9980785846710205, -1.281056523323059, -0.6096484065055847, 1.768268346786499, 0.5686972141265869, 1.280068039894104, 0.5515840649604797, 1.9395571947097778, -0.9027075171470642, 0.20734359323978424, 0.06280988454818726, -0.7966102957725525, 0.6843359470367432, 0.7770445346832275, -0.5779528617858887, -0.19291549921035767, 0.8564052581787109, 2.578190326690674, 0.8918184638023376, -0.11407396197319031, -0.08569281548261642, 0.05963237211108208, 0.12782320380210876, -1.7024316787719727, -0.22849725186824799, -1.2703912258148193, 2.7813141345977783, 0.5115441679954529, -0.195437952876091, -1.4204213619232178, -0.005543229635804892, -0.9580307602882385, 1.3606770038604736, 0.427185982465744, 2.2640979290008545, 0.482179194688797, 0.7476876974105835, 1.2560155391693115, 0.3604925274848938, -0.36969926953315735, 0.5565978288650513, 1.396561861038208, 0.8521838188171387, 0.4042884111404419, 0.5274903178215027, -0.2380257248878479, -0.40790998935699463, -0.5583212375640869, -0.6551564931869507, 0.2793199419975281, 1.1526294946670532, 1.9598660469055176, -0.5500980615615845, 0.13465836644172668, 0.3076757788658142, 0.837113082408905, -0.4925174117088318, -0.7637107968330383, -1.1555935144424438, 0.09079012274742126, 0.9146353006362915, 1.1720131635665894, 0.14253637194633484, 0.10879981517791748, -0.8739116191864014, -1.6100773811340332, -0.16074144840240479, 0.2738804519176483, 0.12386816740036011, -0.4280710816383362, -1.6016714572906494, -0.48988038301467896, -1.9536999464035034, 0.920228898525238, 0.4832751154899597, -0.5699824690818787, 0.45521780848503113, 1.1944152116775513, 0.8343223929405212, -0.0012387189781293273, 1.4586236476898193, 0.29392871260643005, 0.22680054605007172, 0.5154037475585938, -0.5207204222679138, 1.225192666053772, -1.30010986328125, 0.5111879706382751, -1.0419648885726929, 0.2234223634004593, -1.655640959739685, 1.3785443305969238, 0.6134811639785767, 0.3263344168663025, 1.1142033338546753, -0.5381264686584473, -1.1790434122085571, -0.9584968090057373, -0.08451451361179352, -0.07261334359645844, -0.7911033034324646, -1.425277590751648, -0.19263318181037903, 0.46654292941093445, -0.17079047858715057, 1.559680461883545, -0.9366177916526794, 0.13163797557353973, 1.1958699226379395, -0.1024407222867012, -0.361431360244751, -1.0407638549804688, -1.1779332160949707, 2.3185999393463135, 0.3772112727165222, -2.0013580322265625, -0.37034931778907776, 0.28074008226394653, 0.8263309597969055, 2.0346271991729736, 0.6038140654563904, 0.2895261347293854, 1.553541898727417, 1.656503677368164, 0.3366999328136444, 0.943636953830719, 0.26545006036758423, -1.4153728485107422, -0.49423858523368835, 1.3203620910644531, 0.6550856232643127, 1.8441741466522217, -0.3606296181678772, -0.6651570796966553, 1.0492188930511475, 2.166349172592163, -0.19781745970249176, 2.2994372844696045, 0.2909233868122101, 1.3377807140350342, 0.20476889610290527, 0.07114969193935394, -1.2009509801864624, 1.0181872844696045, 0.9340037703514099, -0.16575148701667786, 0.853975772857666, 0.597424328327179, -1.1439014673233032, -0.6123207211494446, -0.05041436105966568, 2.092381000518799, 0.2758951485157013, -0.08396915346384048, 1.0906888246536255, 1.2014472484588623, -0.7621510028839111, 0.4502583146095276, -1.515155553817749, -1.3681124448776245, 1.0919747352600098, 0.5871001482009888, 0.5166094303131104, -0.5385209918022156, -0.3250495493412018, -1.6107786893844604, -0.8145615458488464, 0.06771837919950485, 0.9435153007507324, 0.25527268648147583, 2.0953500270843506, 0.4228673279285431, -0.7166123390197754, 2.0656299591064453, -0.8353182673454285, -0.0952896922826767, 1.3136436939239502, -0.8763113617897034, -1.3582082986831665, 0.03979000821709633, -1.2459707260131836, -1.481086015701294, 1.6112104654312134, 1.271775245666504, -0.2497149556875229, -0.45289212465286255, -0.18293744325637817, 0.31886178255081177, 2.9365668296813965, 0.6573271751403809, -0.7295429706573486, -0.7757906317710876, 0.9810425043106079, 1.848375916481018, 0.6040971279144287, -1.6396536827087402, -1.259878158569336, -0.18379426002502441, -0.44861942529678345, -1.0262681245803833, 2.069817066192627, 0.5133185386657715, 1.3708871603012085, -1.5916774272918701, 0.8707476258277893, -0.12260182946920395, 1.699453353881836, -0.3919979929924011, 0.5047820210456848, 0.4564521908760071, -0.6836415529251099, 0.802017867565155, -0.30468541383743286, -0.11832322180271149, -1.1273130178451538, 0.6158260107040405, -0.09024040400981903, -1.3184584379196167, -0.9805768728256226, 0.012977897189557552, -1.5428822040557861, -0.5882458090782166, 1.4323209524154663, -1.0283650159835815, -0.3682878017425537, 2.4840657711029053, 0.24989016354084015, 0.4410346448421478, 0.5770976543426514, -0.05239572748541832, 0.5565758943557739, 3.415109157562256, 0.9342195987701416, 0.8623833656311035, -0.4077342748641968, -0.9621813893318176, 0.4031555652618408, -0.5195499062538147, 1.3494685888290405, -0.42029258608818054, 0.4046671688556671, -0.10200278460979462, -0.3458522856235504, -0.2899652123451233, 1.1005337238311768, 0.7185208797454834, -1.458242654800415, -0.4843682050704956, -0.7403271794319153, 0.7699030041694641, -0.5092344284057617, -0.6207156777381897, -0.7571115493774414, 1.418491244316101, 0.9176480770111084, -1.998649001121521, 0.001577927265316248, -0.0730392336845398, 0.530267059803009, 0.4782046675682068, 0.9210168719291687, 2.1900808811187744, -1.725205898284912, 1.6634821891784668, -0.46028366684913635, -1.4153612852096558, -0.402493953704834, -0.9656304121017456, 2.5025689601898193, 0.691188395023346, 0.7825290560722351, -0.2802947759628296, 0.13870033621788025, 0.9052432775497437, 0.5153066515922546, 0.6421550512313843, 0.7474918961524963, 0.882912278175354, -0.6716967225074768, 1.2382053136825562, -0.03578570857644081, -0.48688220977783203, -0.960869312286377, 0.9601784944534302, 0.49511104822158813, 1.558869481086731, 0.14995019137859344, -0.4809148907661438, 0.41673097014427185, -0.15665198862552643, -0.5207632780075073, -1.532796859741211, -1.0373741388320923, 0.8347089290618896, 0.46207815408706665, -0.8890491127967834, 0.4038049578666687, 1.3568261861801147, -0.2587059438228607, 0.8120699524879456, 0.40432003140449524, 0.5002527236938477, 0.716754674911499, -1.2078224420547485, 0.7626282572746277, 0.5420669913291931, 2.0019917488098145, -0.11370038241147995, 1.3644217252731323, 0.6369287967681885, -0.6717090606689453, -1.6018190383911133, -0.8665167689323425, -0.27071237564086914, -0.2923479676246643, 0.4954488277435303, 0.6102402806282043, -0.403708815574646, -0.5465627908706665, -1.0229676961898804, 2.687947988510132, 1.5745153427124023, -1.3427331447601318, 0.531234085559845, -0.2974317967891693, 0.045294858515262604, 1.9542498588562012, -0.8880081176757812, 0.1545625478029251, -0.1912636160850525, 1.0533249378204346, 1.486799955368042, -0.29216092824935913, -0.7941312789916992, -0.25303536653518677, -0.4932747781276703, -1.9774162769317627, -1.0933432579040527, -1.0559887886047363, -1.3285820484161377, -1.232085943222046, -1.896150827407837, -0.4191701412200928, -0.8421261310577393, 0.049388643354177475, -1.045980453491211, 0.8411078453063965, -0.831251323223114, 0.8347759246826172, -0.08251623809337616, 0.0985170379281044, -0.922930896282196, -0.744095504283905, 0.12036097794771194, -1.0385324954986572, -0.629209041595459, 0.5666597485542297, 0.2491583228111267, -0.7550061941146851, 0.9682982563972473, -1.2909663915634155, 0.6770596504211426, 0.13940565288066864, 0.29998230934143066, -0.42679741978645325, 0.24707961082458496, 0.7113306522369385, -0.2691993713378906, 0.7095150351524353, -0.01759248785674572, 0.2895350456237793, 1.084572196006775, -0.5549173951148987, 0.15839828550815582, 0.41367244720458984, 0.5186806321144104, 1.4995461702346802, 0.05358552187681198, -0.13222774863243103, 2.609065294265747, -0.9566701054573059, -1.8396281003952026, -0.42527538537979126, 0.7656411528587341, -0.31654664874076843, 1.9592394828796387, 0.47929802536964417, 0.25778117775917053, 1.485703468322754, 0.28510525822639465, -0.47322481870651245, -0.09305433928966522, 0.4114035367965698, 0.2753681242465973, 0.5185953974723816, -0.4970730245113373, 0.806225061416626, -1.225189447402954, 0.26971179246902466, 2.4239718914031982, 1.0800795555114746, 0.6946735978126526, -0.8450813889503479, -1.4253941774368286, 2.0239827632904053, 0.7524868845939636, 0.6757422685623169, 0.6531774997711182, -0.3002924621105194, 1.5327327251434326, -1.373491883277893, -1.2150729894638062, -0.12858149409294128, 0.2818845510482788, 0.5663468837738037, -0.4214020371437073, 0.9082368612289429, -1.0535401105880737, 0.20500977337360382, 1.2868492603302002, 0.07631024718284607, 1.1575618982315063, 2.179572343826294, 1.0058906078338623, 1.393742322921753, 2.2649569511413574, 0.32650554180145264, 0.4616474509239197, 1.2650816440582275, 0.40801718831062317, -1.2021604776382446, -2.2350571155548096, 0.1543550193309784, -0.985848605632782, -0.38171714544296265, 0.543708860874176, -1.1839934587478638, 0.635534405708313, 0.32348310947418213, -1.6688133478164673, -0.42337146401405334, -0.5130093693733215, 0.3367328643798828, -0.5356317758560181, -0.2575266659259796, -0.5085629820823669, -0.4556751549243927, 1.7260878086090088, -0.42421719431877136, 1.5328363180160522, 1.129612922668457, -1.0467288494110107, -1.8338508605957031, -1.2468194961547852, -0.9272571802139282, -0.13371190428733826, 1.6567217111587524, 0.46629253029823303, 0.6775088310241699, 0.9457223415374756, 0.003984094597399235, -0.09687516838312149, -0.14828312397003174, 0.9070615172386169, -0.4964430034160614, 0.35100439190864563, -1.2793829441070557, 1.8268588781356812, -0.1007671132683754, 0.040772344917058945, -1.8839054107666016, 0.7006044387817383, 0.5118806958198547, 3.19730544090271, 0.7655853033065796, -0.32656288146972656, -1.0042922496795654, -1.4512473344802856, 1.4211392402648926, 2.2342212200164795, -0.7722905874252319, 0.010379965417087078, -1.9042227268218994, 0.2769365906715393, 1.4696186780929565, 2.158548593521118, -1.7786836624145508, 0.21923698484897614, -0.27727851271629333, 0.7114446759223938, 0.22355344891548157, -0.7698681950569153, -0.057402513921260834, 1.3467949628829956, -3.168595552444458, -1.3804044723510742, 1.3267384767532349, -0.11094647645950317, -2.0024430751800537, 1.10895574092865, 0.3826999068260193, 0.027544524520635605, 1.3136951923370361, -1.0622386932373047, 2.1706156730651855, 1.3426669836044312, 2.4192070960998535, 0.21989163756370544, -0.9961868524551392, -0.4401063919067383, -1.5266071557998657, 0.8552114367485046, 0.9698328375816345, -0.04851572960615158, 0.08272389322519302, -0.09944097697734833, 0.06852428615093231, -0.4886585772037506, 1.9759690761566162, -2.3207266330718994, -1.0114490985870361, 1.4086737632751465, 1.126257300376892, -0.7498067021369934, 0.09337491542100906, -0.30961164832115173, 0.2472018301486969, -0.6871045827865601, -1.2082130908966064, -0.2265559285879135, 0.6932440996170044, 0.27590036392211914, -1.041487455368042, 0.6431159377098083, -0.5678450465202332, -0.35329723358154297, -2.277086019515991, -0.43539655208587646, 1.0319901704788208, -0.9814316630363464, -0.6951116919517517, 0.33159753680229187, 1.2269320487976074, 1.2862317562103271, -0.054428115487098694, -0.9354372620582581, -0.831019937992096, -0.5673624873161316, -0.21775643527507782, -0.10022284090518951, 2.0700466632843018, 0.949596107006073, -2.0695323944091797, 0.9058615565299988, 0.13280701637268066, 1.052833914756775, 1.258652925491333, -0.651520311832428, 1.4458420276641846, -0.08819780498743057, 1.8132081031799316, 0.4947214722633362, 0.21485233306884766, -0.670640766620636, 0.12056274712085724, 0.3156145513057709, 0.5439673662185669, -0.08944518864154816, -1.1568385362625122, -0.9012612700462341, -0.30589523911476135, 0.8425096273422241, 0.6902452111244202, -0.8575509786605835, -1.3322819471359253, -0.8003285527229309, 0.5351806282997131, -0.05305786058306694, 0.4686426520347595, 0.2976363003253937, 0.2677533030509949, 1.7365341186523438, 0.25009432435035706, -1.3847544193267822, 0.9785599708557129, -1.587251901626587, -0.465847373008728, -2.0319700241088867, -0.08529281616210938, -1.9782284498214722, -1.414717197418213, -0.7325351238250732, 0.15620604157447815, 0.6762981414794922, -1.987525463104248, 0.8949772715568542, -0.587342381477356, -0.28157731890678406, -0.08538012951612473, -1.3440436124801636, 1.3039484024047852, -2.5266189575195312, -1.5205886363983154, 0.30398082733154297, 0.607222318649292, 0.2677883505821228, -0.004004260059446096, -0.8339930772781372, -0.02304704114794731, -0.40758466720581055, -1.7737081050872803, 1.0464844703674316, -0.19193656742572784, 1.1418672800064087, 0.5090693831443787, -0.9075083136558533, -1.444712519645691, 0.7085909247398376, 0.6172110438346863, -0.6424559950828552, -0.6647927761077881, -0.9893300533294678, -0.5133951902389526, 1.7565243244171143, -0.4154697358608246, 1.349226951599121, 0.7432324290275574, -0.9727655649185181, -0.4913175106048584, -1.9932141304016113, -0.5895169973373413, 1.4057847261428833, 0.8849376440048218, 0.28201472759246826, -0.6495364308357239, 0.7839182019233704, -1.5152298212051392, 0.05871141329407692, 0.4626482129096985, 0.7410064935684204, 0.09206590056419373, -0.39341819286346436, 1.3760706186294556, -0.6408334970474243, 0.7944064736366272, 0.21657897531986237, 0.15205101668834686, 2.0878188610076904, -0.11519135534763336, 0.1318839192390442, 0.8947630524635315, 0.7420462369918823, -0.9974195957183838, 1.762589454650879, 0.3625774383544922, 0.22939066588878632, -0.26064857840538025, 1.0032223463058472, -2.019409656524658, -0.33736786246299744, 0.34725862741470337, 0.17104949057102203, -2.754997730255127, 2.0103213787078857, 0.6487640738487244, -0.29012274742126465, 0.28573355078697205, -0.522003173828125, 0.5764575600624084, -0.9007216095924377, -0.5636729598045349, -0.6370089650154114, -0.5776143670082092, -0.3908514082431793, 0.5800476670265198, 0.9464750289916992, 1.4471288919448853, -0.06134037300944328, 0.37008434534072876, 0.8070048093795776, -1.3486145734786987, -1.02614426612854, 0.3804955780506134, 0.48439937829971313, -0.6164748668670654, -1.7422163486480713, 0.1306062936782837, -1.082051157951355, 0.4341941475868225, -0.852980375289917, 1.2431856393814087, -2.954371929168701, -1.0477831363677979, 0.8789801001548767, -0.058035701513290405, 0.2265920341014862, 1.0876483917236328, -0.5408318042755127, 1.2423712015151978, 0.42864328622817993, -0.657089114189148, -0.8082053065299988, -0.08192464709281921, 0.5600500702857971, -1.8956893682479858, 0.5059122443199158, 0.43695947527885437, 1.480212688446045, -1.0015476942062378, 0.6840860843658447, 1.2734928131103516, -0.030145786702632904, 1.631232500076294, -0.0896926075220108, 0.19424007833003998, -1.0686211585998535, 0.7387676239013672, -1.0829346179962158, 1.5974557399749756, 0.2758842408657074, 0.5029420852661133, 0.12781907618045807, -1.3177218437194824, 0.14983265101909637, 1.6911861896514893, -0.7135295867919922, 0.5290101170539856, -1.814234733581543, 0.37800702452659607, 0.46468043327331543, -0.011767067946493626, -0.9758347868919373, -1.0555404424667358, -1.1036275625228882, 0.46759048104286194, 0.6604377031326294, -1.1441859006881714, 0.5485849380493164, 0.7899319529533386, 0.16249248385429382, -0.8464198708534241, -0.0857534259557724, 0.02192661352455616, -1.1789101362228394, -0.8060293197631836, 0.33092525601387024, 0.2921818196773529, -0.384219229221344, 0.4097870886325836, 0.7069198489189148, -1.472474455833435, 1.8459233045578003, -1.0209994316101074, -1.034064531326294, -0.8118294477462769, 0.3223173916339874, -0.16243046522140503, 0.07431419938802719, -0.2574057877063751, -0.7938841581344604, 2.5391154289245605, -1.07546865940094, -0.44857147336006165, -0.7454404830932617, -0.9512653350830078, 1.6701457500457764, -0.40206414461135864, 0.9455814957618713, -0.7203778028488159, -0.9965499639511108, -1.083711862564087, -0.9640212059020996, 0.08995599299669266, -0.560986340045929, 0.11322731524705887, -1.525054693222046, 1.9644688367843628, -0.177425354719162, 0.23666591942310333, 1.2607293128967285, -0.3701816499233246, 0.6618841886520386, 0.45628511905670166, -0.3429390490055084, 1.0118019580841064, -2.0355162620544434, -2.3266639709472656, -0.9696736335754395, -1.637927532196045, 1.2235753536224365, -0.9555890560150146, 0.2789556682109833, -1.4332317113876343, -1.6660008430480957, -0.9196745157241821, -1.3362224102020264, -1.373568058013916, 0.7152064442634583, -1.237244725227356, 1.7173632383346558, 0.017608024179935455, 0.9605769515037537, 1.6932578086853027, 0.4297938644886017, -0.10208636522293091, -0.8039237856864929, 0.45685073733329773, 1.32963228225708, -1.2335599660873413, -2.412729501724243, -0.8495919704437256, -1.3102240562438965, 0.01957736164331436, -0.1600436270236969, 0.09371495246887207, 1.0281180143356323, -0.6395801901817322, -1.4444196224212646, 0.35270294547080994, 1.4000731706619263, 0.5362932682037354, 1.2631163597106934, 0.4299190044403076, -0.23522280156612396, 0.17709839344024658, -1.538080096244812, 0.395616352558136, 0.2860526442527771, -1.0938401222229004, 0.7517059445381165, -0.6941803097724915, 0.1958434283733368, -0.06634578853845596, -0.042712025344371796, 0.80355304479599, -0.8790205717086792, -0.44865164160728455, -1.3146754503250122, -1.107790470123291, 0.35060635209083557, -0.6947764754295349, 1.1052521467208862, -0.8876658082008362, -0.4540957808494568, -1.2927109003067017, 1.1798471212387085, 1.063234806060791, -0.23369050025939941, -1.155189037322998, -0.2987827658653259, -1.8450818061828613, 0.7528938055038452, -0.5220820307731628, -2.620450019836426, 0.6865152716636658, 0.12405981868505478, -0.48628127574920654, -0.24843809008598328, -0.2446059286594391, 1.3883053064346313, 0.12245729565620422, -1.202881932258606, -1.0701886415481567, 1.945983648300171, -0.1554504632949829, 0.3620966374874115, -1.077331304550171, 0.8407852053642273, 1.041019320487976, -0.8732100129127502, 0.6879873871803284, 1.2890766859054565, 0.8312425017356873, -0.8072468638420105, -0.6782631874084473, -0.6647340059280396, -1.6968845129013062, 0.9591719508171082, -0.17034414410591125, -0.5372532606124878, -2.103898525238037, 0.6225899457931519, -0.4298528730869293, 0.19119763374328613, -0.5606945157051086, -1.1551542282104492, 0.33910128474235535, -1.3739210367202759, -0.49919575452804565, 0.1737181842327118, -1.2849634885787964, 0.9851264357566833, -0.05565733462572098, -0.1117057204246521, 0.11728542298078537, -0.27143195271492004, 0.32352012395858765, 0.44037967920303345, 1.1935042142868042, 0.4553147852420807, -1.3481398820877075, -0.47243183851242065, -0.755112886428833, -1.0558106899261475, 0.9064092636108398, 1.8039804697036743, 0.26082155108451843, -0.568863034248352, -0.6075027585029602, -0.18482771515846252, 0.06310541927814484, -0.8179235458374023, -0.030748950317502022, -0.4313257038593292, 1.0894798040390015, -0.5158656239509583, 1.02881920337677, 0.5034249424934387, -0.2751561105251312, 0.45164811611175537], [0.34332630038261414, -2.237513303756714, -1.059824824333191, 0.16422778367996216, -1.451912522315979, 0.8760684132575989, -0.019392888993024826, 0.8005657196044922, -0.2103392481803894, -0.2219727784395218, -0.5955337285995483, -0.9127024412155151, -0.4362586736679077, 1.6808702945709229, 1.5161778926849365, -0.5288918018341064, 0.22802822291851044, 1.3567489385604858, -0.48469650745391846, 1.1370551586151123, 1.1892719268798828, -0.6467359066009521, -0.6952506303787231, -0.06894908845424652, -1.559952974319458, -1.3953887224197388, -0.3179886043071747, -1.1703811883926392, 0.041188787668943405, 1.0138633251190186, 0.9719815254211426, 1.2962347269058228, 0.5000941157341003, 0.3604685962200165, 0.8481230735778809, -1.6810885667800903, 0.17457430064678192, 0.781562089920044, -0.36367717385292053, 0.08791930228471756, 2.3497114181518555, -0.4497333765029907, 0.7081557512283325, -0.15872646868228912, 2.631322145462036, -0.6470786333084106, -1.1691535711288452, -0.7424047589302063, -0.6093016862869263, 0.45294931530952454, 0.595612645149231, 0.20788675546646118, -1.3074135780334473, 0.40678006410598755, 0.13680946826934814, -1.0046058893203735, -0.39626696705818176, -0.26365092396736145, -1.0274081230163574, -0.538334846496582, 0.031779561191797256, 0.5567977428436279, -0.07194771617650986, -0.8174388408660889, -1.2341161966323853, 1.0504463911056519, 2.3383231163024902, -0.9399058222770691, -1.423399806022644, -0.8361213207244873, -2.1223878860473633, -1.219405174255371, -0.37461185455322266, -1.3076913356781006, 0.8372368812561035, 0.017563749104738235, -1.1979146003723145, 0.1349225491285324, 1.037346363067627, 0.06284109503030777, -0.40316393971443176, 0.7895179390907288, -0.2160138636827469, -1.2765111923217773, 0.2981529235839844, 0.5014730095863342, 0.9430687427520752, -1.045159101486206, -1.497078537940979, 0.6278023719787598, 0.6436553001403809, -0.7075942754745483, -0.2797869145870209, -1.248470664024353, 0.24258480966091156, 0.5319145917892456, 3.307068347930908, -0.18452566862106323, -0.07840700447559357, 0.3253057301044464, -1.4065814018249512, -1.1123154163360596, 1.1119111776351929, 0.9139952659606934, 0.12085790932178497, -0.5702418684959412, 0.3648489713668823, 1.853527307510376, -0.6840316653251648, 1.411821722984314, -0.3687422573566437, -1.4955275058746338, 0.14122416079044342, 0.44594553112983704, 0.7116454839706421, -0.8725708723068237, 0.40068674087524414, -0.4976699650287628, -0.7718133330345154, 0.7001223564147949, 1.722827672958374, -1.1169685125350952, 0.6880797743797302, 0.10987222194671631, -0.370084673166275, -0.8395128846168518, 0.6413230299949646, -2.477829933166504, 0.9885197281837463, -0.12094258517026901, 0.13260500133037567, 0.42677074670791626, -1.6468088626861572, -0.6221703290939331, -1.0890100002288818, -0.17121107876300812, -2.5408287048339844, 0.47409847378730774, 1.318595051765442, -0.05642588436603546, 0.8201374411582947, 0.9333596229553223, -0.37474462389945984, 1.1118767261505127, 0.9754140377044678, -0.45582959055900574, -0.7989577054977417, 0.2073967158794403, -0.6955406069755554, 0.4088544547557831, -1.5566155910491943, -1.611438512802124, -0.3385128974914551, 1.323694109916687, -0.036389779299497604, -0.6129022240638733, 2.4315598011016846, 3.528829574584961, 0.9718397855758667, 0.7130903601646423, -0.31776750087738037, 1.360158085823059, -1.4588111639022827, -1.0527328252792358, -0.527474582195282, 1.2919946908950806, -0.5365940928459167, -0.543550431728363, 0.7486366033554077, -0.5328649282455444, -0.8134440779685974, -0.6694383025169373, -0.803180456161499, 0.5121407508850098, -0.48233023285865784, 0.9938973188400269, 0.3133711814880371, -1.7555899620056152, -0.9767817258834839, -0.5604710578918457, 0.1455410271883011, -0.5243033766746521, 0.35339972376823425, -0.41919952630996704, 0.13093231618404388, 0.013046294450759888, 0.497426837682724, 0.3376782238483429, -0.4800977110862732, 0.6924386620521545, 1.0040295124053955, -0.5466835498809814, 1.3865916728973389, 2.2095236778259277, -0.3158194422721863, -0.8011589646339417, -0.27793416380882263, 0.6532722115516663, 0.8808008432388306, 1.955390453338623, 0.6506074666976929, -0.12195152789354324, 0.13614080846309662, 1.1154602766036987, 0.09310662001371384, -0.009890123270452023, 1.1579926013946533, -1.608899712562561, -0.7418040633201599, -0.4180278480052948, 0.6492797136306763, 0.21742428839206696, -0.8602956533432007, 2.4182960987091064, 1.198606252670288, 1.5375475883483887, -0.48317357897758484, 0.22128956019878387, 0.6285306215286255, -1.5273244380950928, -0.6447195410728455, -0.5358469486236572, 0.2924552261829376, -1.7975220680236816, -0.2976298928260803, 0.7336978316307068, 0.350602388381958, 0.025864209979772568, -0.24234506487846375, -0.013528238981962204, -1.2986009120941162, 1.4348582029342651, -0.33633953332901, -0.32756826281547546, 1.0704010725021362, 0.05221842601895332, -0.8614729046821594, -0.3937300741672516, 1.6172245740890503, 1.9748154878616333, 0.8911206126213074, 0.3867020606994629, 0.34326469898223877, -0.6761747002601624, 1.3002102375030518, -1.532394528388977, -1.8669580221176147, 0.153647780418396, -1.890281319618225, 1.2208195924758911, -1.573299527168274, -0.5321100354194641, 0.583662748336792, -0.14806130528450012, -0.28058764338493347, 1.280771017074585, -1.5931198596954346, 0.14328785240650177, 1.0208719968795776, 0.4498213231563568, -0.8676729202270508, 0.03865232318639755, 0.33602970838546753, 0.19033925235271454, -0.05957113578915596, 1.2605831623077393, 1.6896394491195679, 0.10999495536088943, -0.42386943101882935, -1.5813711881637573, -1.5128822326660156, 0.6047285199165344, -1.4800989627838135, 1.8432166576385498, -0.07066812366247177, 0.6215820908546448, 1.3041621446609497, 0.4460142254829407, -1.7833664417266846, -1.2769081592559814, -0.1835552304983139, -0.9105176329612732, 1.3062459230422974, 1.1545429229736328, 0.2783718705177307, 0.48323771357536316, 0.9743365049362183, -0.4718056619167328, 0.4971667528152466, 0.255293607711792, 1.5795985460281372, -0.7381711006164551, -1.4661107063293457, -0.03959493711590767, 2.1334357261657715, 0.9281227588653564, 0.8650692105293274, 0.9103801250457764, -1.0322295427322388, -1.2901161909103394, 0.8745099902153015, -0.43444377183914185, 0.9377829432487488, -2.2465388774871826, 0.24414916336536407, -0.9050953388214111, -0.17995281517505646, -0.5474295616149902, 1.117701768875122, 0.4933069050312042, -0.7297766804695129, 0.08985389024019241, -0.8727943897247314, 1.836561679840088, 1.8767882585525513, 0.9966702461242676, -0.6260085105895996, 0.5121410489082336, -1.3603531122207642, -0.9981343746185303, -0.9373224973678589, -0.10484159737825394, -1.755535364151001, -1.2002109289169312, 0.25352299213409424, 0.1106783077120781, -0.1995353400707245, 0.5446354746818542, -1.289550542831421, -0.8779821395874023, -0.3004191517829895, -0.38266584277153015, 0.15558728575706482, -0.9438402056694031, -1.0069966316223145, -1.05637788772583, 1.269842267036438, 1.878719449043274, -0.04015630483627319, -0.36030399799346924, -0.21167375147342682, 0.28466299176216125, 0.9594053030014038, 0.2284090369939804, -0.8385592699050903, 2.2807140350341797, -1.2093706130981445, -0.2607322037220001, 0.7267858982086182, 1.3745849132537842, -0.12436342239379883, -0.5158444046974182, -1.5213425159454346, 0.3809722661972046, 0.5363365411758423, 1.1998448371887207, 0.593499481678009, -0.1877843141555786, -0.24087470769882202, 1.588781714439392, 1.0108635425567627, -0.5258389711380005, -0.5785853266716003, -0.2658727169036865, 0.44858986139297485, 1.0441868305206299, 0.0065263849683105946, 0.8838797807693481, 0.07566110789775848, 0.25163739919662476, 1.9009733200073242, -1.0433555841445923, -0.44722679257392883, 0.6989074945449829, -1.302620530128479, -2.7068941593170166, -0.42412635684013367, 0.3780129551887512, -1.0135853290557861, -0.4531157910823822, 0.25431039929389954, -0.0022627313155680895, 1.1513378620147705, 0.868599534034729, 0.034963950514793396, 1.1907590627670288, -0.2485470324754715, -0.003867850173264742, -0.2874910235404968, 0.20300930738449097, 0.4121052622795105, -1.9818612337112427, 0.9839039444923401, -1.5177661180496216, -0.4380890130996704, -0.7309471368789673, -1.2019816637039185, -1.3058481216430664, 0.17889223992824554, -1.1800086498260498, 1.8702317476272583, 1.2485157251358032, 0.07292449474334717, -1.0650395154953003, -0.08718843758106232, 0.4062681794166565, 0.24918600916862488, 1.4905787706375122, 1.7514221668243408, -0.15950776636600494, -0.46420523524284363, 0.3796992003917694, -0.25735142827033997, -0.7889536619186401, -1.1153206825256348, 0.7121119499206543, 0.5338298678398132, -0.7507975101470947, 1.749171257019043, -1.404179573059082, -1.3362557888031006, -0.08691322058439255, 0.8239006996154785, 0.7600240707397461, 1.0920915603637695, 0.25492236018180847, -0.9408677816390991, -0.037232305854558945, 0.18458811938762665, 0.1395992487668991, -0.9807082414627075, 0.6104742288589478, -1.42210054397583, 0.868685781955719, 1.4228214025497437, 1.4495633840560913, -2.019503116607666, 1.4276046752929688, 0.4069782793521881, 0.9737915396690369, 0.9415870308876038, -0.9575082659721375, -1.4731248617172241, -0.8632771968841553, -0.3343070149421692, 1.7856075763702393, -0.007986227050423622, -0.1158997118473053, 1.0266656875610352, 0.4294187128543854, 1.710959553718567, 1.1083192825317383, 2.0298070907592773, 1.180672287940979, -0.7929530739784241, 1.3111469745635986, -0.7549211382865906, 0.7086101174354553, -0.13723808526992798, 0.6412848830223083, 0.17905694246292114, -0.5176740884780884, 1.0801043510437012, -0.7753584384918213, 0.4368407428264618, 0.17861586809158325, 0.12632212042808533, 1.0560176372528076, -0.018092593178153038, -0.42180293798446655, 0.012737872079014778, 1.2985765933990479, -1.0174468755722046, 0.5764814615249634, -0.5119466185569763, 1.6597446203231812, 0.20100660622119904, -0.36400315165519714, -0.6032315492630005, 1.2667655944824219, -0.7756661772727966, -0.058954063802957535, -1.1680998802185059, 0.8778153657913208, 0.7391868829727173, -0.7869450449943542, -1.0130404233932495, 0.08356871455907822, -0.7617028951644897, -0.030671793967485428, -1.8664520978927612, -0.5501893162727356, 1.0249371528625488, -0.4075635075569153, 0.3274514377117157, 0.4169093668460846, -1.8009968996047974, 0.07937008142471313, 0.6950836181640625, 0.8415544629096985, -1.095690131187439, 0.9692883491516113, 0.8610222339630127, -0.9917067289352417, -0.9507079720497131, 0.24970215559005737, 1.3666596412658691, 1.4685126543045044, -1.667790412902832, -1.7721452713012695, 1.9588285684585571, -0.685092031955719, 1.4576220512390137, -1.2934919595718384, 0.5948287844657898, -0.0406670942902565, -0.7386672496795654, 0.11668725311756134, -0.14173053205013275, -1.361072063446045, -0.43874824047088623, -1.1738686561584473, -0.37809333205223083, 0.12762142717838287, 0.0307940524071455, -0.8627929091453552, -1.039033055305481, 0.7658175826072693, -0.3541848957538605, 0.6152639985084534, -0.23532983660697937, -0.5985134243965149, 1.1333032846450806, -0.7184875011444092, -1.7619471549987793, 0.13696704804897308, 0.027706556022167206, -0.48499852418899536, -0.12867332994937897, 0.3392477035522461, -1.248504877090454, -0.11146330088376999, -0.5931180715560913, -0.307026743888855, 0.602323055267334, 0.8820555210113525, -0.47432422637939453, -0.5852013230323792, -0.42491456866264343, -0.6993102431297302, 0.6807723641395569, -0.5291487574577332, 0.4310634732246399, 0.6894956231117249, 1.6797987222671509, 0.1686771661043167, -1.422485589981079, 0.13677258789539337, -0.3228055536746979, -0.36326366662979126, 0.5100521445274353, 0.08107808232307434, 0.003097143257036805, 0.12034901976585388, 0.1864095777273178, -1.7504630088806152, 0.5097498893737793, -2.1520562171936035, -0.08675402402877808, 0.9081956744194031, -1.4962947368621826, 0.23104536533355713, 0.6823892593383789, -0.11506091803312302, 0.10868728905916214, -1.8754297494888306, -0.5409042239189148, 0.8580186367034912, -0.7422044277191162, -0.6377457976341248, 0.9455728530883789, -0.02058161422610283, 1.0315901041030884, -0.6222977042198181, -0.9557689428329468, -0.3704316318035126, 0.9614842534065247, 0.9396698474884033, 0.31141164898872375, 0.2168334275484085, -0.04176872968673706, -0.45962101221084595, -0.25423455238342285, -0.648429274559021, 0.11849750578403473, -0.41990113258361816, 1.02908194065094, 1.7188118696212769, 1.2843674421310425, -0.7027727365493774, 0.21219024062156677, 0.6177045106887817, 0.25331011414527893, 0.04009369760751724, -0.2584506869316101, 0.31448987126350403, -1.31321382522583, 1.8269027471542358, -0.36584755778312683, 0.21941202878952026, -0.8919336795806885, 0.008113263174891472, -1.1716059446334839, -0.914462149143219, 0.4024178981781006, 1.425479531288147, -1.9711004495620728, 0.5423641204833984, 0.9673171639442444, -0.1480688452720642, -0.31139692664146423, 0.36738061904907227, -2.193605899810791, -1.175093412399292, -0.6229929327964783, -0.6143980622291565, -0.30201953649520874, 1.9650418758392334, 0.4062734544277191, 0.31225937604904175, 0.23202210664749146, -0.785639226436615, -0.7267228960990906, 0.2482701539993286, -0.6397513151168823, -0.20572222769260406, 0.5954484343528748, 1.0049912929534912, 0.6303618550300598, -0.15544380247592926, -1.8112177848815918, -0.08431033045053482, -0.610505998134613, 1.3820477724075317, -1.002764344215393, -0.3821150064468384, -0.12685993313789368, -0.26181018352508545, 0.7068834900856018, -0.2748553156852722, -1.3973668813705444, 0.3999783396720886, -1.0656541585922241, 2.102020502090454, 0.2243661880493164, 0.9309046268463135, 0.9099971055984497, -0.4681757688522339, -1.4191139936447144, -0.5799571871757507, -0.9369567632675171, 1.8738549947738647, 0.50248122215271, -0.7697539925575256, 1.647542119026184, -0.28818127512931824, 1.195356011390686, 1.5266300439834595, -0.4387710690498352, -0.5386307835578918, 0.38145211338996887, -0.03725652024149895, -2.0911405086517334, -0.32423532009124756, 0.2557377517223358, 0.1402565836906433, -1.2675825357437134, -0.44098684191703796, -0.8066791892051697, -0.47780826687812805, 1.5368211269378662, -0.8994844555854797, 0.9604796767234802, -0.19126982986927032, -1.8255406618118286, 1.1765798330307007, 0.47965362668037415, -1.0614815950393677, 2.856492519378662, -0.4623071551322937, 0.16295450925827026, -0.9816649556159973, -1.5867594480514526, -1.7091710567474365, 0.823349118232727, -1.1634879112243652, -2.8948938846588135, -0.04888860508799553, -1.4519178867340088, 2.129768133163452, -0.06523151695728302, -1.2471503019332886, 0.310505211353302, 0.5146949291229248, 0.19838176667690277, 0.06230616196990013, -0.21423520147800446, -0.8185576796531677, 1.6128861904144287, 0.08621980994939804, 0.6961609721183777, 1.0384680032730103, 0.3163009285926819, 1.4661657810211182, 0.9725133776664734, 0.3398389518260956, -0.9935257434844971, 0.01757833920419216, -0.9280043244361877, 0.2566588521003723, 0.006290087942034006, 0.793188750743866, -0.2636363208293915, -0.7335187792778015, 0.08613049983978271, 0.24998484551906586, -0.4497363865375519, -0.09719624370336533, 0.06824911385774612, 0.7042654156684875, 0.20156218111515045, 0.44439804553985596, -0.6841959357261658, 1.6008466482162476, 0.3801097571849823, -1.4299277067184448, -1.0158226490020752, -0.26452335715293884, 2.1083967685699463, -2.3310670852661133, 0.9664334058761597, 0.49770796298980713, 1.1334431171417236, -1.2109217643737793, -1.151513695716858, -0.9068671464920044, 0.7499064803123474, -1.727036476135254, 0.30199652910232544, 1.024549961090088, 0.8730040788650513, -0.9331547021865845, -0.6081079840660095, -0.8015663027763367, -0.8454265594482422, -0.2597219944000244, -0.052512191236019135, 0.8324127197265625, 1.1074148416519165, 0.42534133791923523, 1.8156555891036987, 1.8484320640563965, 0.8532565236091614, 0.02120324783027172, 0.9635723829269409, 0.4630810022354126, -0.9243270754814148, -0.23374594748020172, 0.22841162979602814, 0.4793497323989868, -1.5732426643371582, 0.7925270795822144, -1.8864308595657349, 1.0581129789352417, -0.9952488541603088, 1.2075121402740479, 1.0465373992919922, -1.975879192352295, -0.8863224387168884, -0.9781476855278015, -1.6321923732757568, 1.3729653358459473, 0.2553272545337677, 2.3160452842712402, 0.3213981091976166, -0.793694257736206, 1.545899510383606, 1.131661295890808, -0.8947827219963074, 0.38667821884155273, -0.7816440463066101, 1.0034520626068115, 1.3802292346954346, -0.14022989571094513, -1.0244386196136475, 0.3404681384563446, 0.18059098720550537, 0.1888856142759323, -0.5744048953056335, -0.054248709231615067, 0.6110314726829529, -0.9287493824958801, 0.40919971466064453, 1.192594051361084, 0.4621008634567261, -1.2104697227478027, -0.03250144422054291, 0.2578594982624054, 0.5821136832237244, 1.48576819896698, 1.40876042842865, -0.021233022212982178, -1.2201546430587769, -0.5510356426239014, -0.16041982173919678, -0.06117064505815506, 0.108762226998806, -0.5855676531791687, -0.9740939736366272, 1.4633264541625977, 1.2488782405853271, -0.08567828685045242, 0.9267759323120117, -1.2832865715026855, -1.6592172384262085, -0.1657532900571823, -1.8420648574829102, -0.4218984544277191, 1.6514898538589478, -0.6953881978988647, 0.7750011682510376, 2.282696008682251, 0.34044745564460754, 0.02255183830857277, -0.9331250190734863, 0.583118200302124, -0.7031092643737793, 1.4380542039871216, 1.1947853565216064, -1.0199851989746094, 0.6107383370399475, -1.1056432723999023, -0.24792182445526123, -1.6146012544631958, -0.10149551928043365, -0.2840699255466461, -0.15901246666908264, -1.2612966299057007, -0.9550930261611938, 1.6315999031066895, 1.8037562370300293, -1.7719764709472656, -0.5200251936912537, 0.5421499609947205, -0.5071723461151123, 1.1541757583618164, -0.24104362726211548, 0.9087359309196472, -0.4688453674316406, 1.1817420721054077, 1.1407570838928223, 0.813116192817688, 0.8401073813438416, 1.5736862421035767, -0.3516611158847809, 1.0403729677200317, -0.19171138107776642, 0.7889676094055176, 0.28827691078186035, 0.5142888426780701, 0.7039763331413269, 1.1231313943862915, 1.1959468126296997, -1.3292733430862427, 1.0196198225021362, 0.2927229106426239, 0.06393762677907944, 0.9530213475227356, 0.5444532036781311, 0.8035867810249329, -0.14869754016399384, 2.3994221687316895, -0.16774842143058777, -0.7510901689529419, -0.35124000906944275, -0.6121975779533386, -1.4361135959625244, 0.2314695417881012, 1.026140570640564, -0.4655202627182007, 0.6778016686439514, -0.002082856371998787, 1.9409751892089844, -1.586652159690857, -2.1097419261932373, -0.0396401509642601, -0.36160847544670105, -1.5236515998840332, -0.4080081880092621, -0.6137614846229553, 0.14574547111988068, 0.7156928777694702, -1.1601805686950684, 0.25434610247612, -1.761545181274414, -2.208338975906372, -1.107034683227539, -1.27534019947052, 0.6476524472236633, 1.1107622385025024, 0.43099695444107056, 1.8663742542266846, -0.17237460613250732, -0.13525550067424774, -1.342667818069458, -0.5181965827941895, -0.18435901403427124, 0.6984262466430664, 0.33316516876220703, -1.5093674659729004, -1.3234643936157227, -0.21263980865478516, 1.6427102088928223, 1.4245615005493164, -0.06745031476020813, -1.3318772315979004, 0.1998588740825653, -1.8500932455062866, 2.074140787124634, 0.14136560261249542, -0.8283829092979431, -0.45121994614601135, -0.2777835726737976, -0.41345393657684326, -1.079904556274414, -2.5526435375213623, 0.6567199230194092, -1.6762055158615112, 0.6821097731590271, -3.434413194656372, -0.7933648228645325, 0.770254373550415, 1.2026371955871582, 0.876156747341156, -1.597853422164917, 0.26666927337646484, -0.5881232023239136, 0.22182147204875946, -0.643878698348999, 0.9021304845809937, -0.3150310814380646, 1.2741552591323853, -0.3992007374763489, -1.3048101663589478, 1.509685754776001, -1.090980052947998, -0.17689891159534454, 0.33262789249420166, 1.6733653545379639, 0.5160186886787415, 1.447144865989685, 0.34202808141708374, 2.4283230304718018, -0.14252477884292603, -0.1804867535829544, -0.38010185956954956, 0.06188822537660599, -0.6290266513824463, 0.5059839487075806, -1.043280005455017, -0.5378124117851257, 0.2532344460487366, 0.2425423562526703, -1.7270170450210571, -0.544111430644989, -1.5276916027069092, 1.4911330938339233, 0.4353446960449219, 0.25224360823631287, 0.19007809460163116, -1.4924192428588867, 0.0681936964392662, 1.4592419862747192, 0.17044447362422943, -0.05107564479112625, 0.39671313762664795, 0.10100313276052475, 1.218953251838684, -0.5427513122558594, 0.8208662867546082, 1.565873384475708, -0.6780679225921631, -0.620526909828186, -0.6763135194778442, 0.5325592756271362, 0.2719283401966095, 0.713837206363678, -0.32190829515457153, 0.7044205069541931, -0.09035498648881912, -0.8815639615058899, -1.2832486629486084, 1.768228530883789, -0.5398097634315491, -0.1029670462012291, -1.108109474182129, -0.26968011260032654, -0.0429844856262207, 0.8352716565132141, -0.8258975744247437, 0.2867407500743866, 0.39665669202804565, 0.7045859694480896, 0.9051215648651123, 2.4339513778686523, -0.7212652564048767, 0.4629737436771393, -1.4266120195388794, 1.6866782903671265, -0.9444665312767029, 0.9308026432991028, 0.30428463220596313, -0.32164266705513, 1.5275218486785889, 0.0060068476013839245, 1.41054368019104, 0.7780486941337585, -1.1011948585510254, 0.06684133410453796, -0.6441468000411987, 1.6344152688980103, 0.6384726762771606, 0.5231096744537354, -0.9133279323577881, 1.0565383434295654, -1.6532626152038574, -0.0201357901096344, 0.036638978868722916, -0.8186224699020386, 0.46459129452705383, 0.13672584295272827, -0.03786469250917435, 0.1610075682401657, 1.4376486539840698, -0.5350673794746399, 1.9744617938995361, 1.6000641584396362, -2.5596306324005127, -0.3430550992488861, 0.07393714040517807, 1.0366770029067993, -0.4918062686920166, 0.29433590173721313, -0.3073717951774597, 1.4067471027374268, 0.05225382745265961, -0.7457257509231567, 2.291016101837158, -0.14144934713840485, -1.0391565561294556, 1.1795125007629395, -0.7641708850860596, -0.2683502435684204, 0.5731388926506042, -0.1047341376543045, -0.37106582522392273, -0.3971250057220459, 0.14622020721435547, -0.25975966453552246, -0.7641364336013794, -0.6514923572540283, -1.4644709825515747, 0.7534087300300598, -1.457329511642456, 0.17347154021263123, -0.9916740655899048, -0.45114198327064514, -0.2796045243740082, -0.6965388059616089, -1.2536978721618652, 0.4442002475261688, 0.10628251731395721, -1.2785183191299438, 1.595048189163208, 1.3104496002197266, -0.9453328847885132, 1.318542718887329, 0.8676844239234924, 0.2761768698692322, -0.3898976147174835, 0.018173208460211754, -0.1528216004371643, 0.7619000673294067, 0.760869026184082, -0.9145705103874207, 0.09119506180286407, 0.29882729053497314, 1.0342379808425903, -0.4958884119987488, -1.708611249923706, 1.3366529941558838, 1.2459803819656372, 0.7480933666229248, 0.44069692492485046, -0.5962326526641846, -0.18800339102745056, -0.05820360779762268, 1.8333208560943604, -0.22282084822654724, 0.25028476119041443, -0.5526272654533386, -0.008144774474203587, 0.04198890179395676, 1.0216387510299683, 0.31919535994529724, 1.9367728233337402, 1.2011585235595703, 0.5230276584625244, 3.0744917392730713, -0.9184698462486267, 1.023849368095398, -0.0900760293006897, 1.9905723333358765, -1.3235470056533813, 0.14348618686199188, -0.03691073879599571, -0.3056270182132721, -0.9639861583709717, 0.618524968624115, 3.1350643634796143, -0.5697854161262512, 1.8615647554397583, 1.038910984992981, 1.0064821243286133, -1.078514575958252, -1.3072108030319214, -0.34726959466934204, -0.6539438962936401, -0.8753255009651184, -0.4716312885284424, -1.4565696716308594, 1.239782452583313, 0.5183275938034058, -0.830653190612793, 0.632491946220398, -1.1268292665481567, 0.7822237014770508, 0.2225051075220108, 0.09549178928136826, 1.0685315132141113, 0.18499936163425446, -1.8804094791412354, -1.631935954093933, -0.9460850954055786, -1.3262450695037842, -2.103498697280884, 1.0128248929977417, -0.06202005594968796, -0.3668227195739746, -1.6146272420883179, 1.3159589767456055, 1.1783968210220337, 1.149720311164856, -0.3687534034252167, -0.5389077067375183, 1.0658416748046875, -0.8173742294311523, -0.35089290142059326, -0.16697104275226593, -0.18182608485221863, -0.1623455137014389, -0.1841413676738739, 0.09723378717899323, 0.9425711035728455, -1.198117971420288, 1.4920741319656372, 0.43130993843078613, 0.19982211291790009, -0.003001444973051548, 0.8017074465751648, 0.4224807918071747, -1.8194594383239746, -0.6801663637161255, -0.46091076731681824, -0.11189742386341095, -0.5356150269508362, 1.024847149848938, -1.212708592414856, 0.2107287347316742, -0.8152457475662231, -1.8021029233932495, 2.994077205657959, -1.1814318895339966, 0.7793825268745422, 1.0985314846038818, 1.6668331623077393, -0.8285068869590759, 0.6806812882423401, 0.4061490297317505, 0.46450942754745483, 1.1859068870544434, 0.6710227131843567, 1.2947876453399658, 2.16196870803833, 0.10444353520870209, 0.061735376715660095, 0.33377066254615784, -0.1748441755771637, -0.9644829034805298, 2.2726292610168457, 1.6423649787902832, 1.6481995582580566, -2.4552807807922363, -0.09551464766263962, 1.3507498502731323, -0.22171303629875183, -0.5971683263778687, 1.1269049644470215, -0.3904440701007843, 0.10140503942966461, 0.30957505106925964, 0.6420164108276367, -0.6146764755249023, -0.6559422016143799, -0.21855071187019348, -1.7012540102005005, 0.24784672260284424, -1.3394526243209839, 1.7800772190093994, -0.760896623134613, -1.0217031240463257, -0.3072025775909424, -1.5379942655563354, -2.3003602027893066, -1.4216867685317993, -1.2114893198013306, -0.9674428105354309, 0.5583925247192383, 0.20586885511875153, -1.0135517120361328, 1.2198351621627808, 0.37353062629699707, 1.6339092254638672, 0.7739909887313843, 2.242436170578003, -0.06418519467115402, -0.3726367950439453, -0.40451669692993164, 0.31893789768218994, 0.5627668499946594, 0.5187840461730957, 1.3496959209442139, 0.042489875108003616, 0.6078184843063354, -0.8969681262969971, -1.4870364665985107, 0.7832269668579102, -1.5610607862472534, 0.6537774801254272, 0.520420491695404, -1.609732985496521, 2.4180266857147217, -0.39905408024787903, 0.4979914426803589, -1.4178717136383057, 1.0012540817260742, -1.1367344856262207, -0.4222886264324188, -0.18659715354442596, -0.4652784764766693, 1.0475698709487915, -0.23504306375980377, 0.8458778262138367, 1.4219392538070679, 0.7962400913238525, -0.3986041843891144, -0.427092969417572, -0.5405299663543701, -0.3182154893875122, -1.0738499164581299, 0.23413068056106567, -0.6309589743614197, 1.0760669708251953, 0.9105316400527954, 0.17373597621917725, -1.537339210510254, 0.7050666213035583, -1.163631558418274, -1.430677056312561, 0.49277135729789734, 1.1544398069381714, 0.2961527407169342, -0.005983870942145586, -1.4541898965835571, -0.3611430525779724, 0.9962747097015381, 1.5767160654067993, -0.9625060558319092, 1.4521191120147705, 1.0380758047103882, -0.18784596025943756, -1.3415679931640625, 0.15596111118793488, -0.8966043591499329, 0.6576118469238281, -0.5521356463432312, -2.196378707885742, 0.2528221905231476, 0.13167858123779297, 0.518277108669281, -0.32819586992263794, 1.2852064371109009, 1.0601235628128052, 0.8336626887321472, 0.12068000435829163, 0.560757577419281, -1.5938116312026978, 0.8291298151016235, -0.2360559105873108, -0.6388548016548157, -0.9117546081542969, -0.7371165752410889, 1.9857107400894165, 0.06668556481599808, 0.5705189108848572, 2.401618719100952, -0.12949907779693604, -2.781791925430298, 1.6404612064361572, 2.3094892501831055, -0.3011646866798401, 1.1350812911987305, 0.07221196591854095, -0.26226693391799927, 1.1354318857192993, -0.9196610450744629, -1.6536638736724854, 0.590477705001831, -0.05046628415584564, 0.22158175706863403, 1.6041903495788574, -0.3330256938934326, -0.3530896306037903, -2.9873740673065186, -0.11667291074991226, 1.0236154794692993, 1.068709373474121, -1.0912210941314697, 0.3980361521244049, -0.05562558025121689, -1.680303692817688, 0.7982980608940125, -1.6954008340835571, -1.6867648363113403, 0.3623642027378082, -0.18207815289497375, 0.35449984669685364, 0.5363336205482483, 0.4019959270954132, 0.525301992893219, 0.7176845073699951, 0.9105768799781799, -0.11359797418117523, -1.71170175075531, -0.7026820778846741, 0.1165928915143013, 0.5397525429725647, 0.31732770800590515, -0.32734131813049316, -0.08297678828239441, -0.39814865589141846, 0.517853319644928, 0.11897871643304825, 1.0554914474487305, -1.409364938735962, 1.6227351427078247, 1.0251048803329468, -0.43984559178352356, 0.7829867601394653, -0.5951837301254272, -1.296255350112915, -0.9111493825912476, 0.9922612309455872, -1.1123822927474976, -1.5802221298217773, 0.41920262575149536, 0.4433356821537018, -2.0693984031677246, -0.07238779962062836, 0.26995357871055603, 0.6374499201774597, 0.8837157487869263, 1.596191644668579, -0.03708783909678459, -0.5885818004608154, 0.32070064544677734, 1.3286277055740356, -1.282273292541504, 0.2366531938314438, -0.5840683579444885, 0.8563435673713684, 0.8625076413154602, 0.3578561544418335, -0.8343645930290222, -2.2626760005950928, 0.08538288623094559, -0.3409937620162964, 1.1386947631835938, 0.40529757738113403, 0.434198260307312, -0.13912369310855865, 0.19147808849811554, 1.3724455833435059, 0.4640050530433655, 0.2901831567287445, 1.8259186744689941, -0.037598736584186554, 1.0260001420974731, 0.45356348156929016, 0.14953193068504333, 0.05223812907934189, -0.5988346338272095, -0.4638032615184784, 0.19426389038562775, -0.1535218060016632, -2.137840747833252, 1.2671864032745361, 1.1106367111206055, -2.3913378715515137, -0.3702276945114136, -0.3587433695793152, -1.1154441833496094, 1.2767425775527954, -0.7290704846382141, 1.0774078369140625, -0.8804188370704651, -0.8175428509712219, -0.8680834770202637, -0.334532767534256, -2.6016616821289062, 1.004515528678894, 0.06520017981529236, -0.5232352018356323, -0.2577544152736664, -0.4901672303676605, 0.0931566134095192, 1.8634059429168701, 0.9967399835586548, -1.2918699979782104, 1.473862886428833, -1.6581223011016846, -0.9838210940361023, -1.1610305309295654, 0.3318641781806946, 0.06835155189037323, 0.8825656175613403, -0.38020750880241394, 1.001534104347229, -0.24187465012073517, -0.12487836182117462, -2.4208271503448486, -0.1580500602722168, -0.4867594540119171, 0.36490046977996826, 1.3645581007003784, -1.3133325576782227, 1.648919939994812, 0.5853059887886047, -1.753037452697754, 1.1394706964492798, 0.956709623336792, 0.47800952196121216, -1.1879730224609375, -2.746236562728882, 1.733197569847107, 0.18318521976470947, -0.23719261586666107, -1.5074694156646729, -1.1547213792800903, 0.22349825501441956, 0.6228684782981873, -1.4737945795059204, 0.06310858577489853, 0.38033172488212585, 0.1360091120004654, 0.3793131709098816, -0.7035945653915405, -1.291097640991211, 1.1311572790145874, 0.5465928316116333, -1.267225742340088, 0.09820303320884705, -0.5142577886581421, -0.16957448422908783, -0.04311954230070114, -0.7349546551704407, -0.37946704030036926, -0.8056362271308899, -0.23368270695209503, -0.5551243424415588, 0.39014673233032227, -0.6341574192047119, -0.3414420783519745, -0.9433866143226624, -0.5694776177406311, -3.0020108222961426, -0.9545837044715881, 0.1915583610534668, 0.9298793077468872, -0.6897842884063721, 0.5543429851531982, -1.245985507965088, 0.21582959592342377, -1.117598295211792, 0.38790252804756165, 0.6602153182029724, 1.848840594291687, 1.174672245979309, -1.2817836999893188, -0.013699453324079514, 0.4692634642124176, -1.304472804069519, -1.3939770460128784, 0.8693221211433411, 0.03797200694680214, 0.8638244867324829, 0.14298468828201294, 1.2079190015792847, -1.589555025100708, -0.5273646712303162, 0.8303282856941223, -0.2597329914569855, 0.7975735664367676, -0.7145100831985474, -0.4983837902545929, -0.3032228350639343, -0.313001424074173, -0.32492759823799133, -1.1734404563903809, -0.6114861369132996, 0.43268540501594543, 1.2922977209091187, 1.378328800201416, -1.5684062242507935, -1.2228790521621704, 1.3099777698516846, 1.0543404817581177, 0.4469919204711914, 0.9437980651855469, -0.2682647705078125, -0.14193646609783173, -0.09300311654806137, 0.5262744426727295, 0.069663405418396, 0.8701474070549011, 2.106555461883545, -2.2786691188812256, 0.28223949670791626, -0.2407664805650711, 0.20035333931446075, 1.0095465183258057, -0.7301974296569824, -0.6070351004600525, 1.4067468643188477, -1.8006057739257812, 0.21675533056259155, 0.797027051448822, 0.8099176287651062, 1.1459752321243286, -1.2113405466079712, -0.33385634422302246, 2.587196111679077, -1.4024462699890137, -0.7022313475608826, 0.12366187572479248, -1.3181806802749634, -0.9105517864227295, -0.6122756600379944, 1.0683974027633667, 0.6579379439353943, 1.6352812051773071, -1.4561500549316406, -0.2578212320804596, 0.5176888704299927, 0.49835070967674255, 1.8817615509033203, 0.33354562520980835, 0.08625344932079315, 0.6296485662460327, -0.3407914936542511, 1.9996001720428467, -0.46940210461616516, -1.2440975904464722, 0.07807338982820511, 0.27612024545669556, -1.2809520959854126, 0.12886278331279755, -0.9777433276176453, -1.5716748237609863, 1.1941193342208862, 0.0813637226819992, 0.16866832971572876, 1.6199842691421509, 0.3265606164932251, 0.4816930890083313, 0.6607546210289001, 0.24393272399902344, -1.475319504737854, 1.7030380964279175, 0.5864272713661194, 1.0446425676345825, 0.1824311912059784, -2.406872510910034, -1.582694172859192, 0.3504353165626526, 2.9571006298065186, 0.8432015776634216, -0.06450317800045013, -0.6065747141838074, 1.2854727506637573, 0.35936251282691956, 0.363194078207016, 0.8789202570915222, 0.031078290194272995, 0.3449046313762665, -0.7629031538963318, -1.626629114151001, -1.4503086805343628, -0.5558302402496338, -0.07424352318048477, -1.3335438966751099, 0.5883551239967346, 1.771371841430664, 1.8126487731933594, -0.7160277366638184, 2.2382588386535645, 0.554109513759613, -2.1010165214538574, 1.9791793823242188, 0.2537839114665985, 0.7259758114814758, -0.16945870220661163, 1.3978016376495361, 0.9018415808677673, -0.3370632231235504, 0.35229966044425964, 0.24828988313674927, 1.342862844467163, -0.24415190517902374, 2.674999713897705, 0.303425133228302, -0.1826297640800476, 1.6565091609954834, 0.8740322589874268, -0.09789200872182846, -0.9144032001495361, 0.0805509015917778, -0.3725161552429199, 1.3871852159500122, -0.6901635527610779, -0.6684143543243408, 0.320119708776474, -0.9536540508270264, 0.9718284606933594, -0.06099022552371025, -0.3045903444290161, -0.3179505467414856, 0.8319730758666992, -0.7614962458610535, -0.21794746816158295, -0.6499282717704773, -1.3171418905258179, 0.06730321049690247, -1.2669970989227295, -1.5061650276184082, 1.8444105386734009, 0.7154802083969116, -1.6431331634521484, -0.3082028329372406, -0.6992613673210144, -0.37463143467903137, -1.2103121280670166, 0.400236040353775, 0.5302126407623291, -0.833453357219696, -1.5757606029510498, -1.049613118171692, 0.4104706645011902, -0.17267543077468872, 0.7093575596809387, -0.27467793226242065, 1.3592091798782349, 1.1768649816513062, -0.5601552128791809, 0.7629804015159607, -1.2820088863372803, -2.105262041091919, -0.9834287762641907, -0.11862102895975113, 0.39353495836257935, 1.3883222341537476, 0.06897832453250885, 0.041732098907232285, -1.2170255184173584, -0.904789388179779, -0.5453431010246277, -1.0413925647735596, -0.49634280800819397, -1.4266471862792969, 0.6248086094856262, 0.04968151077628136, -0.8223483562469482, -0.022484302520751953, -0.5839098691940308, 0.32368162274360657, -0.2640451192855835, -1.6643157005310059, -0.11741887032985687, -0.32375311851501465, 0.3724479377269745, 0.5712710618972778, -1.0922709703445435, -1.0461808443069458, -0.30721044540405273, -0.713114857673645, 0.22313860058784485, 0.9073557257652283, 0.12937380373477936, -0.16228710114955902, -0.10717631131410599, -0.14303776621818542, -1.1710118055343628, 0.9179673790931702, 0.4871784746646881, 1.6959118843078613, -1.2002719640731812, 0.023427044972777367, 0.6425220370292664, 0.5263959169387817, -0.21577182412147522, -1.2636756896972656, -0.5600476264953613, 0.5926744341850281, 0.16665968298912048, 0.8886277675628662, -0.11570221930742264, 0.23489077389240265, 0.5863562226295471, 0.016266049817204475, 1.260223627090454, -1.36863112449646, -0.982019305229187, -0.09376528859138489, -1.2988461256027222, 0.7753946185112, -0.5722933411598206, 1.750960111618042, -0.280659019947052, 1.4986090660095215, 0.8164154291152954, -0.7347835302352905, -1.5964553356170654, 0.2056199461221695, -0.6820572018623352, -1.907793402671814, -0.8076481819152832, 1.184080719947815, 0.3147408068180084, 0.5551926493644714, 1.0221045017242432, -0.7960774302482605, 0.8819816708564758, -0.852273166179657, 0.6545324921607971, 0.5457341074943542, 0.19726988673210144, 0.11515823006629944, -0.8855268955230713, 0.8437747359275818, -1.754945158958435, 0.44681888818740845, 1.5396347045898438, 1.2498445510864258, -0.3153110444545746, -0.2616727352142334, 0.7595290541648865, 1.1778490543365479, 0.5160330533981323, -1.706202507019043, 1.426989197731018, -0.7300624847412109, -2.556645154953003, -0.7833219170570374, 0.22125144302845, 1.2547723054885864, -0.8998352289199829, -0.7282143235206604, 0.8130623698234558, 0.6857404708862305, -0.2673923075199127, 0.5983384251594543, -0.712547779083252, -0.6802738904953003, 0.4380520284175873, 0.3551519215106964, -0.3570747673511505, 1.418447732925415, 0.8867322206497192, -0.28456419706344604, 1.0550830364227295, -0.37216824293136597, -0.30137744545936584, -0.025782546028494835, 0.026065731421113014, -0.9620532393455505, 0.7837603092193604, 0.8497492074966431, 0.3814266324043274, -0.22734831273555756, 1.013028860092163, 0.03673022612929344, -1.3850351572036743, -0.6148124933242798, 1.1476850509643555, -1.9883708953857422, 1.673431634902954, -0.13651706278324127, -1.2531810998916626, 0.23647871613502502, -1.487964391708374, 0.05712675303220749, 1.8839876651763916, 0.1037585511803627, -0.5950869917869568, -1.4861128330230713, 0.8136155009269714, -0.8154070377349854, 1.8470901250839233, 0.2893613278865814, -0.07859578728675842, -0.7887651920318604, 2.111349105834961, -1.7168729305267334, 1.7751843929290771, 1.3089219331741333, -0.6457392573356628, 0.5711799263954163, 1.6054681539535522, 0.336366206407547, 0.45539259910583496, 0.4684036076068878, 0.7265892028808594, -0.5878994464874268, 0.2647170126438141, 1.235851526260376, -0.17511603236198425, 0.43323373794555664, 1.536647081375122, 0.7970078587532043, -0.3011741638183594, -0.784691572189331, -0.32540300488471985, -0.25710824131965637, -2.0879766941070557, 2.1269936561584473, 0.6375712752342224, 0.07740578055381775, -0.6360117793083191, -0.022296186536550522, 1.5033644437789917, -0.0228470116853714, -0.46630939841270447, 0.6562613844871521, 0.7714165449142456, 0.20267419517040253, -1.3870593309402466, -0.6302675008773804, 0.9364315867424011, -1.3390357494354248, 1.6007286310195923, 0.9794428944587708, -0.21526718139648438, -1.8798155784606934, 0.48049867153167725, 0.02001374028623104, 0.8507155776023865, 0.6402411460876465, 0.2988283336162567, 0.8833431005477905, -0.9493131637573242, 0.17431652545928955, 0.9665601253509521, 0.40177983045578003, -0.7077051997184753, -0.40410125255584717, -0.2899380326271057, 1.0633448362350464, -1.399428129196167, -0.7190297245979309, 1.1116588115692139, -0.1678813248872757, 0.29503586888313293, -0.6618505120277405, 0.11503797769546509, 0.6809932589530945, 0.6777240633964539, -0.17022883892059326, 1.2504760026931763, 0.5146632194519043, 1.158334732055664, 1.0813463926315308, -0.7435720562934875, -0.02419050969183445, -0.6947861909866333, -0.14036889374256134, 0.9874545335769653, 0.3942696750164032, 1.3066891431808472, 1.0763628482818604, -2.012021541595459, -0.0797276645898819, -0.6240717172622681, -0.23470032215118408, -0.27648770809173584, -1.2880593538284302, -0.9909791350364685, 0.5523430109024048, -1.080369234085083, -0.4594239592552185, 0.24104230105876923, -1.3701084852218628, 0.3477051258087158, 1.9765472412109375, 0.13316650688648224, -0.6267193555831909, 1.4267834424972534, 0.42572736740112305, -0.1731584519147873, -1.731040120124817, 0.7760815620422363, 0.44293293356895447, -0.6621655821800232, 1.5502793788909912, -1.0433961153030396, 0.7585140466690063, 0.5953267216682434, 0.39731118083000183, 1.234818935394287, 0.6078479886054993, -0.6389455795288086, 0.538700520992279, -0.5604064464569092, 0.1557198464870453, 0.6480018496513367, 0.018357349559664726, 1.06276273727417, 3.1521835327148438, -1.4995886087417603, 0.36414095759391785, 0.11779791116714478, 0.6352689862251282, 1.2511359453201294, 1.3301206827163696, 0.9510499238967896, -1.1574594974517822, -0.5777828097343445, -0.2821771204471588, 1.136481523513794, -0.3323161005973816, -1.413956880569458, -0.33393594622612, 1.4773045778274536, 0.8247146606445312, -0.38124850392341614, 0.7533679008483887, -0.6121211647987366, -1.941614031791687, 0.7203217148780823, 1.1254011392593384, 0.47734081745147705, -0.6947845816612244, 1.238873839378357, -0.3814084231853485, 0.8183791041374207, 0.8659387826919556, -0.5989102721214294, -0.9168335199356079, 0.8678990602493286, -0.3119088113307953, -0.6869131922721863, -1.592572808265686, -0.3188476860523224, 0.6604070663452148, 0.5531916618347168, 1.0310051441192627, -0.9017234444618225, 0.5533932447433472, 1.4110509157180786, -0.4869906008243561, 0.5627912878990173, -0.33625921607017517, 1.4023401737213135, -0.3214492201805115, -0.0937841385602951, -0.4843360483646393, 0.14460577070713043, -0.36933404207229614, -0.4215954840183258, -0.05904386565089226, 0.771244466304779, -0.40968745946884155, -0.006000400986522436, 0.6800788640975952, -0.22780728340148926, 0.16649356484413147, 0.5167058110237122, 1.1625028848648071, 0.8257225751876831, -0.10646692663431168, -1.2017117738723755, -1.226083755493164, 0.3488365113735199, 0.5758621096611023, 0.039903659373521805, -0.35298433899879456, -0.037517692893743515, -1.5478074550628662, 1.423354148864746, -0.5596521496772766, 1.055357575416565, 0.12307899445295334, -0.24539713561534882, -1.8629047870635986, -0.9096259474754333, 1.209916114807129, 0.335763543844223, -0.9835864901542664, 1.1726092100143433, 0.03252759948372841, -0.5777198672294617, 1.0421440601348877, 1.9714635610580444, -1.315950870513916, -0.6130015254020691, 0.5034904479980469, 1.4604419469833374, 0.7708515524864197, 0.881818413734436], [-1.9142708778381348, -1.131056547164917, 0.31709298491477966, -0.2310781478881836, -1.9189915657043457, 0.15591509640216827, -0.7963938117027283, 0.5800853967666626, 0.35944852232933044, 0.9996299743652344, 0.652900755405426, -1.0602229833602905, -0.4788065254688263, 1.0997880697250366, 1.510570764541626, 0.9359272122383118, -0.05283264443278313, 0.9163591265678406, 0.8984438180923462, -0.13465580344200134, -0.43544653058052063, -0.0549369677901268, 0.8288304805755615, -0.5646138191223145, 0.9916839599609375, 0.5509043335914612, -0.7195543646812439, -0.9338099360466003, 1.224754810333252, -1.2413421869277954, 0.6560066342353821, -0.9207565188407898, 2.183170795440674, -1.1321436166763306, -1.8768047094345093, 1.404712438583374, -0.565127968788147, 1.6120538711547852, -0.8842899799346924, -0.3176690638065338, -0.6720612645149231, -0.9459179043769836, 1.15729820728302, 2.0708868503570557, 1.5313129425048828, 1.5387928485870361, -0.39564597606658936, -0.851947009563446, 1.1122732162475586, -0.6265205144882202, 0.2754293382167816, 1.2601091861724854, -0.29218924045562744, 0.5055432915687561, -1.2005295753479004, 2.6802995204925537, 0.5757300853729248, 0.8272939920425415, -0.8091000318527222, 1.3117661476135254, -1.059975028038025, 1.3897148370742798, 0.5487431883811951, -0.6055760979652405, 1.2652865648269653, 0.038365431129932404, -0.15633603930473328, 1.1883578300476074, -0.2610250413417816, -0.33911991119384766, 0.4534607529640198, 0.9783763885498047, -1.9373035430908203, 0.9327855706214905, -1.18179190158844, -1.154726266860962, -0.1270184963941574, 0.4714844822883606, 1.3400150537490845, -0.3387499153614044, -1.4218411445617676, 1.4946290254592896, 0.2974594533443451, -0.6490864753723145, -0.3580435514450073, -2.3770391941070557, -0.31633245944976807, 2.366741418838501, -1.2567263841629028, -0.9316021203994751, -1.610097885131836, -0.18745672702789307, 0.09368136525154114, 2.002044916152954, 0.7457029819488525, 0.4122615456581116, -0.4869983196258545, 0.1270418018102646, 0.10423535853624344, -1.5177052021026611, -1.353089451789856, 0.9228256940841675, -0.4528493583202362, 0.3607040047645569, -0.18700793385505676, -0.5442680716514587, 0.4170994460582733, 0.24004587531089783, 0.060002248734235764, -2.412205219268799, 1.2831356525421143, -0.3143470883369446, 0.9062225222587585, 0.42708107829093933, -1.4028276205062866, 0.7405298352241516, 3.231405735015869, -0.3179253935813904, 0.6550998091697693, -1.6344043016433716, -0.587749183177948, 1.3160412311553955, -0.10118156671524048, -0.026858875527977943, 1.5773632526397705, -0.7765863537788391, -0.6663081645965576, 0.21378283202648163, 0.5528550744056702, 0.7232478857040405, 0.0898611843585968, 1.5010267496109009, -2.6362831592559814, -1.6870008707046509, 0.7169203758239746, -0.7407386898994446, 1.182233214378357, -0.9204899072647095, 0.871205747127533, -0.21875235438346863, -0.37529247999191284, -0.08314746618270874, -0.23401932418346405, -0.34473615884780884, -0.11683113127946854, -0.7135004997253418, -0.5959487557411194, 0.800861656665802, 0.40671777725219727, -1.283526062965393, -0.9005337953567505, 1.0284388065338135, -0.13524244725704193, -0.8954892754554749, 0.7211978435516357, -2.257361888885498, 0.971329391002655, 0.24790342152118683, -0.21622616052627563, -1.21665620803833, 0.5310683846473694, -0.9964318871498108, 0.2367180436849594, 0.5506390333175659, 0.682191789150238, -0.2613368332386017, -1.0765763521194458, -0.06328795105218887, 1.6133567094802856, -0.29458996653556824, 0.37713998556137085, 0.2992821931838989, -3.796128273010254, 0.5873667001724243, -0.5351608991622925, -1.1247378587722778, 0.2963579595088959, -0.06619663536548615, -2.1599180698394775, 2.1010608673095703, 1.1250165700912476, -0.5866014957427979, 0.47792601585388184, 1.8632633686065674, 0.042940959334373474, 0.10390926897525787, 0.5450133085250854, 0.492822527885437, -0.6550780534744263, 1.3369181156158447, -0.5589997172355652, -0.17110463976860046, -1.3208290338516235, 0.1179712563753128, -0.04250757396221161, 1.2270866632461548, -0.7805486917495728, 0.006124765146523714, 0.5884509086608887, 0.9012270569801331, 0.5632621645927429, 0.08847042918205261, -2.150580644607544, -1.9906076192855835, 1.0800983905792236, -0.7007091045379639, -0.6785680651664734, 0.3132975101470947, -1.0617084503173828, -1.4168941974639893, 0.5172839760780334, -0.8646658658981323, -2.369537115097046, 1.117794394493103, 0.7169893980026245, -0.5065641403198242, -0.05507320910692215, -0.5664376616477966, 0.530622124671936, -0.8639910221099854, -1.0838775634765625, -0.722601056098938, -1.6761058568954468, 1.8199150562286377, -1.3195966482162476, 0.14995482563972473, 0.9242004156112671, -0.6258484721183777, -0.19712670147418976, 0.2910219430923462, 0.8554686307907104, -0.47927069664001465, -1.2469781637191772, -0.6584759950637817, -0.4936024248600006, -0.31306782364845276, 0.46936002373695374, -0.5718149542808533, 0.8841913342475891, 0.6637921333312988, 0.4229534864425659, 0.5442662239074707, -0.9134788513183594, -0.8626371026039124, 0.5300256013870239, -0.30508744716644287, -0.1243838295340538, 0.6660592555999756, -0.7434502243995667, -1.7345341444015503, 0.9509907960891724, -0.3049444854259491, -0.8993411660194397, 0.8342685103416443, 0.33472684025764465, -1.0097318887710571, 0.7206521034240723, 1.3656821250915527, -0.3364941477775574, 2.3016297817230225, -0.07297953963279724, -0.5191083550453186, 0.7684839963912964, -0.23764073848724365, -0.4616992175579071, 0.47570696473121643, -0.16495604813098907, 1.0746124982833862, -0.4050976634025574, -0.728900134563446, 1.1974033117294312, 0.5646694898605347, 1.152269959449768, 0.591838538646698, 0.3462950587272644, 0.35320621728897095, 0.8227770924568176, 0.9186062216758728, 1.1224780082702637, -0.19809207320213318, 0.23701614141464233, -0.19399309158325195, -0.45307600498199463, -0.5401926636695862, 2.175853967666626, 0.5925467610359192, -0.40277099609375, -1.2141305208206177, 0.3790830671787262, 0.2371828854084015, -1.1532957553863525, 0.04652547463774681, 0.39159122109413147, -0.935980498790741, -3.4195268154144287, -0.7783276438713074, 0.9586763978004456, -0.3033188283443451, -0.5034858584403992, -0.4088052809238434, -1.061012864112854, -1.30872642993927, 0.6801431775093079, -0.29987815022468567, 0.25065863132476807, -1.005630612373352, -0.978875994682312, 0.27266186475753784, 1.027618646621704, -1.4699047803878784, -0.8754902482032776, -0.21584706008434296, -1.0485517978668213, -0.3075098693370819, 0.6847823262214661, -0.38214001059532166, 0.2676149606704712, 1.3596779108047485, 0.7158641219139099, 0.018826821818947792, 1.3573219776153564, -1.2676278352737427, 0.5669350028038025, -0.564635694026947, -1.8510507345199585, 0.14465589821338654, 1.1890333890914917, -0.5643054246902466, -0.6518920660018921, -0.42107152938842773, 0.3383129835128784, 1.4226293563842773, 2.1124794483184814, 0.34162381291389465, 0.37133488059043884, 0.7392247915267944, -0.19267895817756653, 1.2970144748687744, -0.0007418998866342008, -0.4792928695678711, 0.7629393935203552, -1.0014405250549316, -1.9008070230484009, 0.5051237344741821, -0.14217469096183777, 1.7071765661239624, -0.22805961966514587, -0.8774808049201965, -0.8543511629104614, 1.2685333490371704, 0.5699112415313721, 0.8025690317153931, 0.3467733561992645, -0.4625310003757477, 0.13938121497631073, 1.0673044919967651, 0.009735943749547005, -1.3689261674880981, -0.8717821836471558, 1.1084721088409424, -0.21528668701648712, 0.48900699615478516, -0.6578599810600281, 1.8190258741378784, -0.1596522331237793, 0.9318041801452637, 0.20898987352848053, -0.09755484759807587, -0.2696986794471741, -0.8317515850067139, 0.3600549101829529, -0.1572021245956421, 0.4350324869155884, -0.9382907748222351, 0.9248392581939697, -0.5405714511871338, 0.663489043712616, 0.08576899766921997, -0.8783810138702393, 1.0956734418869019, -0.03717953711748123, 1.0334199666976929, 1.4346297979354858, 0.1867595911026001, 0.06520240008831024, 1.1794687509536743, 1.7993887662887573, -0.3241618871688843, 0.7708720564842224, -1.0598008632659912, -0.1179068461060524, -0.05224992334842682, 1.1367238759994507, -1.8912761211395264, -0.397006630897522, -0.6050639152526855, -1.313294529914856, -0.7439013719558716, 0.6389541625976562, -0.22870710492134094, -0.8154279589653015, -0.237987220287323, -0.33935093879699707, -0.34427782893180847, -1.426101565361023, 2.8258445262908936, -1.453682541847229, 1.0386676788330078, 0.9789103865623474, -0.8824158906936646, -0.13890598714351654, 0.45608723163604736, 0.4584144651889801, 0.32862070202827454, 0.6910845637321472, 0.33688685297966003, -0.08699052780866623, 0.33609506487846375, 0.787449061870575, -0.015727059915661812, 0.00030430388869717717, -0.6235755681991577, -0.5072054862976074, -0.5530118346214294, -1.2557893991470337, 0.3409232795238495, -0.5970140099525452, -1.003907322883606, 0.46970975399017334, 0.3687141239643097, -1.0638254880905151, 0.6162404417991638, 1.744296669960022, -0.043695852160453796, -0.5724344849586487, 1.69865083694458, -0.18670323491096497, -0.6825187802314758, 0.3863455355167389, -0.631030797958374, -1.7591840028762817, 1.2915581464767456, -0.7182534337043762, 1.167640209197998, 0.5445126295089722, -0.6316275596618652, -0.1314316987991333, 0.7264617085456848, 1.2542823553085327, -1.4744460582733154, -1.825236201286316, -0.8123754858970642, -2.2022523880004883, 0.4167517423629761, 0.4630860686302185, -0.6633385419845581, 0.7541704773902893, -0.8356393575668335, -1.6684391498565674, 0.6931713223457336, 0.20967088639736176, 0.7698760628700256, 0.027349557727575302, 0.34979239106178284, -0.15737047791481018, 1.6598974466323853, 0.4356936812400818, -1.1639212369918823, -0.5042012929916382, 0.954843282699585, 0.12713763117790222, -1.6699414253234863, -1.6406174898147583, -0.7573917508125305, 0.12546919286251068, -0.7312861680984497, -0.40498748421669006, 0.4063093364238739, -1.794649362564087, -1.2739611864089966, -0.7860648036003113, 1.1362392902374268, -0.02314949408173561, -0.7240130305290222, -0.13739928603172302, 1.186254620552063, 0.16521328687667847, 0.4951043426990509, -0.45784977078437805, 0.24897165596485138, -0.5623672604560852, 0.46099382638931274, 0.09506288170814514, -0.2075948864221573, 0.16737554967403412, 1.3137645721435547, 0.08491158485412598, 0.7946645021438599, -0.7806464433670044, -0.9453462362289429, -1.1259911060333252, -1.3770766258239746, 0.039480604231357574, 1.6587295532226562, 0.7611364722251892, 0.07760230451822281, -0.6614425182342529, -1.2442700862884521, -1.2912230491638184, -0.774968683719635, 0.7951268553733826, -0.47513148188591003, 0.621014416217804, -1.7388097047805786, 0.5405077934265137, 0.2934224605560303, 2.1838841438293457, 1.1059781312942505, -1.6337424516677856, 0.42036351561546326, 0.23245865106582642, -0.6233985424041748, 1.8583325147628784, -1.2669744491577148, 1.0051172971725464, -1.011534571647644, 1.822772741317749, 2.0940394401550293, -0.16196216642856598, 0.09688513725996017, -0.08101072907447815, -0.028896937146782875, 1.088681697845459, 0.7243900895118713, -1.4337284564971924, 0.04752311110496521, 0.282770574092865, -0.41302454471588135, -0.2104167342185974, -0.33484452962875366, 1.291075587272644, -0.4125676453113556, -0.9651832580566406, 0.7428275942802429, 0.5808039903640747, -0.15016192197799683, -0.1867961585521698, 1.0570446252822876, -0.46323809027671814, 0.4685203433036804, -1.298671841621399, -0.8127569556236267, 0.04192233830690384, -1.3688000440597534, 0.8954690098762512, -0.7828088998794556, -0.5938727855682373, 1.1585378646850586, 0.555385947227478, -0.7416732311248779, -0.5718936920166016, -0.9967753291130066, -1.4281002283096313, 0.3944275975227356, -2.296191930770874, -0.13092948496341705, 2.480778694152832, 0.00937803927809, -0.10645434260368347, 0.03454617038369179, -1.224420189857483, 1.5940983295440674, 0.17378591001033783, 0.32462558150291443, 0.6556276679039001, -0.8744834065437317, 1.087053894996643, 0.2402440458536148, 1.1769087314605713, -0.48044076561927795, -1.858299970626831, 0.4814138114452362, -0.5464009046554565, 0.25589364767074585, 1.398821473121643, 0.6799665093421936, 0.8068190217018127, -0.12812156975269318, 1.1502472162246704, -0.497259259223938, 0.057436421513557434, -0.6358470916748047, 0.3143462538719177, -0.5981429219245911, 1.336112380027771, -2.910595178604126, -1.9185256958007812, 1.483359694480896, 0.6472976803779602, 0.8150180578231812, -1.8142367601394653, 1.6209834814071655, -1.6064931154251099, 0.4645630419254303, -0.30409756302833557, 0.49259236454963684, -1.239939570426941, 1.4165078401565552, -0.5658133625984192, 1.3657933473587036, 1.350447654724121, -0.7697905898094177, -0.5205593705177307, 0.6101698875427246, 0.41053590178489685, 0.6456888318061829, -0.14784975349903107, -1.0629605054855347, 0.41271182894706726, 0.22208444774150848, -1.2999502420425415, 2.3093624114990234, 0.21197056770324707, 0.14978040754795074, -1.1371835470199585, 0.8348326683044434, -0.4265329837799072, 0.7225421071052551, -0.7833658456802368, -0.5465114712715149, -1.4585597515106201, -1.8693231344223022, -0.7498079538345337, 1.02175772190094, -0.6542026996612549, 0.7260746955871582, 1.6368470191955566, -1.0926427841186523, -0.5789416432380676, -1.1428831815719604, -0.4159550666809082, 0.05717363953590393, 0.06095374748110771, -1.2667796611785889, -0.8235353827476501, -0.017936713993549347, 0.6851959228515625, -0.21468974649906158, 1.1596848964691162, -0.2313126027584076, 1.867737054824829, 0.804222822189331, -0.28572481870651245, -1.4774067401885986, -0.7780601382255554, -0.9662696123123169, -0.5282831192016602, -0.3872033357620239, -0.7176831960678101, -0.46479445695877075, 1.1502541303634644, 0.011258424259722233, 1.3044761419296265, 0.22888480126857758, -0.4603496193885803, -0.23310954868793488, -0.04982210323214531, -0.40767234563827515, 0.6301769614219666, 0.2797031104564667, -1.0291290283203125, 0.16117772459983826, 0.9221733212471008, -0.963502049446106, -0.14907164871692657, -0.3532659411430359, -0.9577596187591553, -0.14164769649505615, 0.1710994690656662, -0.5508795380592346, -0.8242307901382446, 0.07438656687736511, -0.6753793954849243, 1.0996723175048828, 1.4774186611175537, -1.0404927730560303, -0.3689360022544861, -0.09176385402679443, -0.79105144739151, 0.6624357104301453, 0.9388899207115173, 0.05735677108168602, -0.24668580293655396, -1.0374606847763062, 0.6609968543052673, 1.0611517429351807, 0.0364978052675724, 0.5350053310394287, 1.7225308418273926, 1.304391860961914, 1.525091528892517, -1.5798985958099365, 0.9415698647499084, 0.2581172287464142, -0.4648073613643646, -0.7068620324134827, -1.085550308227539, 0.6220383644104004, 0.4423298239707947, -0.13246524333953857, -0.6100234389305115, 0.37505173683166504, -1.137473225593567, 0.11762914806604385, 1.0396132469177246, 0.9193304777145386, -0.7488795518875122, 0.6142353415489197, 0.1898288130760193, 2.284269332885742, -1.410205602645874, -0.6592365503311157, -0.24832436442375183, -1.5011729001998901, -0.10456378012895584, 1.659895896911621, -0.4375573694705963, -0.7042743563652039, -1.2938175201416016, -0.11584465205669403, -1.4559751749038696, -0.8243614435195923, -0.2091761827468872, -0.5350777506828308, 1.3532207012176514, 2.470367431640625, -0.8538341522216797, 1.6744673252105713, 0.2486070990562439, 0.32473430037498474, -0.018460093066096306, 0.9214181900024414, -1.9096717834472656, -0.02088078111410141, -0.18441534042358398, 0.13037419319152832, -1.9419838190078735, -0.3427313566207886, -0.25098663568496704, -1.7611721754074097, -0.7449833750724792, 0.45466962456703186, 1.3667736053466797, -0.011151377111673355, 1.2611342668533325, -0.17598046362400055, -0.6501916646957397, -1.5779156684875488, -1.1117970943450928, 0.02026316709816456, 0.4709668457508087, -1.774856686592102, 1.3235788345336914, 2.084425687789917, -0.3908185362815857, 1.817741870880127, -0.45214948058128357, -0.6425482034683228, -0.5268386602401733, -1.6162679195404053, -1.272858738899231, 0.9224086999893188, 1.1655421257019043, -1.0838881731033325, -0.9510537981987, -0.07145760953426361, -2.8686211109161377, 0.6096055507659912, 0.3812163472175598, 0.10946682840585709, 0.020866980776190758, -0.8005186319351196, -0.3941730260848999, -0.20127162337303162, 0.1844910979270935, -0.7255547642707825, 0.5831700563430786, 1.0475867986679077, -0.5059872269630432, 0.8228595852851868, 0.7446781992912292, -0.992426872253418, 0.19281652569770813, 1.1120160818099976, -1.8938955068588257, -2.0164999961853027, -0.32928910851478577, 0.03885873407125473, 1.376495361328125, -0.29156383872032166, 0.64356929063797, -0.45034530758857727, -0.19138188660144806, -1.8733218908309937, 0.23154662549495697, 2.699387788772583, 0.3675132393836975, -0.2986302375793457, 0.17928986251354218, -1.2250514030456543, 0.78133624792099, -2.244682550430298, -0.5398250818252563, -0.39353513717651367, 0.15279620885849, -1.2093509435653687, -0.2660815417766571, -0.24120968580245972, 0.30578070878982544, 0.2533973157405853, 0.34259340167045593, 1.157002568244934, 0.1426077038049698, -0.09935097396373749, 2.196768045425415, 0.7033994197845459, 1.6635546684265137, 0.06872010231018066, 1.0155115127563477, -1.3929638862609863, 0.606810986995697, 0.25964751839637756, 0.05664458125829697, -0.6602460145950317, 0.19666609168052673, 0.2229488044977188, 0.4831317067146301, 0.347403347492218, -0.2772282063961029, -0.45054319500923157, -0.6344149708747864, -1.2919961214065552, -0.2663239538669586, -1.0824344158172607, -0.6106227040290833, 0.16219544410705566, -0.16910581290721893, -0.2761465311050415, 0.322605699300766, 0.7337551712989807, 0.4074523150920868, -0.15675576031208038, -0.411211222410202, 0.6333428025245667, -0.38539186120033264, 0.2998281419277191, 0.6861863136291504, 1.2498805522918701, 0.9797131419181824, -0.8797544240951538, -0.9408249855041504, 0.4478611946105957, -0.7270683646202087, -0.12050484120845795, -0.42627763748168945, -0.4766732156276703, -1.1842772960662842, 0.4976818561553955, 0.590864896774292, -0.7393174767494202, -1.4632422924041748, 0.0831553116440773, 1.0172630548477173, 0.6171589493751526, -0.7539089322090149, 0.26561975479125977, 0.02111857570707798, 0.09602415561676025, 0.6775422692298889, 0.06238197162747383, -0.9033994078636169, 1.7058347463607788, 1.2327429056167603, -1.4410361051559448, 0.2382631003856659, -0.1976814866065979, 0.04918989911675453, -0.8591734170913696, 1.108860731124878, -0.26582613587379456, 1.18869948387146, 1.9083340167999268, -0.417589396238327, 1.320346713066101, 1.0766500234603882, -0.9385587573051453, 0.20758166909217834, 0.508083164691925, -0.40641582012176514, -0.5072333812713623, -0.015267174690961838, -0.39637765288352966, -0.89179527759552, -0.6613811254501343, -0.1840333193540573, 0.6350536346435547, 0.633111298084259, -0.436620831489563, 2.1551387310028076, 0.4239952862262726, 0.33648914098739624, 0.13950054347515106, 0.9276120066642761, 1.1616419553756714, -0.2214919477701187, -0.6174706816673279, 0.22128058969974518, 0.6172005534172058, 1.2190132141113281, -1.1601035594940186, 1.0296826362609863, -0.17790056765079498, -0.6343262791633606, 0.5705606341362, 0.47946926951408386, -0.9419483542442322, 0.264333575963974, -0.10688595473766327, 0.31637513637542725, 0.3810849189758301, -1.273847222328186, 0.5411757826805115, 1.320395827293396, 0.18559251725673676, -1.0766875743865967, 3.406439781188965, -1.225156307220459, 0.05413852259516716, 0.021474329754710197, -0.6041176915168762, -0.9785410165786743, 0.912360668182373, -1.3809672594070435, 0.5669946074485779, -0.058140356093645096, 0.4403674900531769, 0.26680269837379456, -0.5612789988517761, 0.8231506943702698, -1.2180769443511963, 1.6034727096557617, 0.1429423987865448, -0.18557685613632202, 0.5675695538520813, 0.08819542080163956, -0.8807116746902466, 1.105555772781372, 0.3642804026603699, 0.4108317792415619, -0.25539833307266235, 1.3549695014953613, -0.33896589279174805, -1.1004852056503296, 0.2203514575958252, 0.616239607334137, 1.4024425745010376, -0.4693309962749481, -0.8717530965805054, 0.24513377249240875, 1.595371961593628, -0.9895150661468506, 1.3675379753112793, -0.33431917428970337, -0.30406054854393005, -1.864997386932373, -0.18348859250545502, -1.077336311340332, -0.10822208225727081, 2.757517099380493, -0.6730745434761047, -0.8908140659332275, 1.058756709098816, 0.20571193099021912, -0.748084545135498, -2.37158465385437, -0.42417415976524353, -1.821166753768921, -0.5453492999076843, 0.6052178740501404, -0.5566860437393188, -2.960569143295288, -0.29697945713996887, -0.32180115580558777, -0.2535223364830017, -0.40341755747795105, -0.024142296984791756, 1.5832184553146362, 0.691897451877594, -0.039411015808582306, -0.9790681600570679, -0.9333101511001587, -1.7188138961791992, -0.8231210112571716, -0.13800735771656036, 0.30783209204673767, -0.08952421694993973, 0.9948164820671082, -1.2444968223571777, -0.840172529220581, -0.4399797320365906, 0.0324232317507267, -0.13480791449546814, 1.2098021507263184, 1.4553577899932861, -0.6790584325790405, -0.8022281527519226, 0.8083984851837158, -1.7698155641555786, -0.34748533368110657, -1.0133496522903442, 0.17889979481697083, 1.8985681533813477, 0.056158509105443954, 0.10628510266542435, 0.8904393911361694, 1.0401109457015991, -1.0824311971664429, 0.2573838233947754, 0.8019766211509705, 0.11338098347187042, -0.06515061855316162, -0.2244643270969391, 0.14874202013015747, 2.1350882053375244, 0.29270821809768677, 1.1254315376281738, 1.96269690990448, 1.456159234046936, 1.445817232131958, 0.10710828006267548, 0.44848036766052246, -1.829849123954773, -1.1233247518539429, 0.43208056688308716, -1.4274511337280273, -2.0520179271698, -1.3324862718582153, 1.4708571434020996, 0.7331412434577942, 0.11778988689184189, 0.6359370946884155, -0.0017629398498684168, -1.1854304075241089, -0.4237265884876251, -0.6064761281013489, -0.10712864995002747, -0.025699328631162643, -0.08337485790252686, 0.07562631368637085, -0.9596686363220215, -0.47057044506073, 1.5453672409057617, 0.44536420702934265, 0.35257744789123535, -0.14954833686351776, -0.3515663146972656, 0.23839767277240753, -0.5263075828552246, -0.06680559366941452, 1.6466683149337769, 0.8745429515838623, 0.1190791055560112, -0.9940581321716309, -0.5151300430297852, -0.45369791984558105, 0.8206127882003784, 0.6646284461021423, 1.2769122123718262, 0.9640631675720215, -0.618678867816925, -0.027096442878246307, 0.05725843459367752, -0.12228448688983917, -0.5944858193397522, 1.3361445665359497, -0.11904773116111755, 1.7475838661193848, -0.45386210083961487, 0.2084892839193344, 0.7068601250648499, 0.7628512382507324, -0.41880977153778076, -1.1091864109039307, 0.5341867208480835, 0.23187361657619476, -1.7736128568649292, -1.038036584854126, -0.3547735810279846, 0.01686149835586548, 1.6442934274673462, -1.6062127351760864, -0.36167436838150024, 0.09175044298171997, -2.7625107765197754, -1.2682225704193115, 1.4875121116638184, -0.8132073879241943, -0.11271577328443527, -0.5209541916847229, -1.420894980430603, -0.38217341899871826, 0.808875560760498, 0.9195802211761475, -0.9940801858901978, 1.9655673503875732, -0.9607212543487549, -0.980033278465271, 0.1736745536327362, 1.2728488445281982, 0.07440691441297531, -0.7468841671943665, 1.4751782417297363, 0.007184675429016352, -1.196104645729065, -0.4933563470840454, 0.38754791021347046, -0.22472085058689117, 1.7583903074264526, -0.4737129211425781, -0.15728886425495148, 0.25676077604293823, 0.13250312209129333, 0.6896044015884399, 0.45189622044563293, -0.8393117189407349, 1.5737684965133667, -0.21740375459194183, -0.6431839466094971, -0.10907960683107376, -2.0835814476013184, 1.5473064184188843, -0.9974302053451538, -0.3096682131290436, -1.2960652112960815, -0.1046871691942215, 0.3896608054637909, 1.374869465827942, 0.21016202867031097, -0.793384850025177, -0.5047683119773865, 1.0940622091293335, 0.0482989177107811, -0.34091684222221375, -1.331976294517517, 0.3957041800022125, -0.22345462441444397, 0.3407447338104248, -0.08807606995105743, -0.4077220559120178, 0.43353158235549927, -1.142859935760498, 1.0915862321853638, 0.1359536498785019, 0.06571650505065918, 1.970926284790039, -0.9069674611091614, 0.21865476667881012, -0.2639535367488861, 0.2643337845802307, 0.20986774563789368, -0.18554659187793732, -0.12572342157363892, -0.6901867389678955, 1.4749224185943604, -0.344788521528244, 0.23154553771018982, -0.9371770024299622, 1.492581844329834, 1.5642826557159424, 0.2807478904724121, 0.25069189071655273, -0.7990207076072693, -1.5088061094284058, -0.9064769744873047, -0.7337611317634583, 1.1710745096206665, -1.4761470556259155, 1.080642819404602, -0.4428839683532715, 0.6778095960617065, -0.6563632488250732, -0.5213088393211365, -0.43748682737350464, 0.3281900882720947, 0.2104220986366272, 0.5351279973983765, -1.1681513786315918, -0.724986732006073, 0.9787254929542542, 0.9435457587242126, -0.4577346742153168, 0.07839915156364441, -1.415364146232605, 0.23437443375587463, -0.5142624378204346, 0.8255515694618225, 1.7823947668075562, -1.0730737447738647, 0.7178482413291931, 1.548096776008606, -2.483130931854248, -0.6086662411689758, -0.3149486184120178, 0.570705771446228, 0.5050882697105408, 0.5282494425773621, -1.7564789056777954, 0.9470260739326477, 0.2422587275505066, 1.9869805574417114, -0.5246948599815369, -0.3980537950992584, -1.191323161125183, 0.38842809200286865, 0.26654350757598877, 2.2875328063964844, -1.6166781187057495, 1.4503778219223022, 0.5625510215759277, -0.7458345890045166, 0.8844667673110962, 0.76407390832901, 0.23334647715091705, -0.0670296773314476, -0.011167248710989952, -0.4055514931678772, -1.427673101425171, -0.882297158241272, -0.1111922487616539, -0.1643037497997284, -1.0835769176483154, 0.5281749367713928, 1.1644959449768066, 1.3761038780212402, -0.1047777310013771, -1.3648828268051147, -0.25277161598205566, 0.4585738778114319, 1.1474770307540894, 1.0384995937347412, -1.6313248872756958, -0.8182098865509033, -0.6075314283370972, 0.2650294303894043, -0.7875741720199585, -0.758694052696228, -0.27233511209487915, -1.439318060874939, -0.3591504693031311, 1.1200541257858276, -1.284462571144104, -0.7516294121742249, 0.7036461234092712, -0.945103645324707, 1.2754786014556885, -2.6444034576416016, 1.0610557794570923, -0.29174238443374634, 1.9857763051986694, 0.40611910820007324, 1.2943748235702515, 0.6364192962646484, -0.528372049331665, -1.5885438919067383, -0.8517211079597473, 1.0867422819137573, -1.2218458652496338, -1.236280918121338, -1.7646735906600952, 1.8200581073760986, -0.049812737852334976, 0.9602116942405701, 0.32197120785713196, -0.3454008400440216, -1.2316832542419434, -0.17663200199604034, 2.294323444366455, -0.0159295704215765, 0.645119845867157, 0.15976881980895996, 0.38329073786735535, -0.21144431829452515, -0.6235925555229187, 1.4803752899169922, -0.36215484142303467, 0.9639556407928467, 2.0488550662994385, -0.33255481719970703, 0.3763676881790161, 1.1403251886367798, -0.42695352435112, 1.9832117557525635, -0.7456503510475159, -0.520025908946991, 0.7291544675827026, 0.17832478880882263, 1.0037068128585815, -0.653934895992279, 0.442466139793396, 1.0144864320755005, -0.6474853754043579, 0.9904415607452393, -0.7553857564926147, 1.6812806129455566, -0.6356151700019836, -0.38950735330581665, 0.1121583953499794, 0.14937521517276764, 0.24665868282318115, 0.2739843726158142, 0.411575049161911, 0.06943514943122864, 0.49293649196624756, 0.5525807738304138, 0.8231247663497925, 0.1730591207742691, 0.004481966141611338, 0.43122023344039917, -1.6987934112548828, 0.7221194505691528, -1.1246824264526367, -0.32648929953575134, 0.1439858376979828, -0.9066378474235535, 0.2640754282474518, -0.16488075256347656, 0.5526269674301147, -1.0761688947677612, -0.6272647976875305, 0.2838788330554962, -1.3202433586120605, 0.1414494812488556, 1.0913461446762085, 2.260366678237915, -0.08441147953271866, -0.3673866391181946, 1.3137972354888916, 1.4371768236160278, -1.422951340675354, 0.14672236144542694, 0.4946065843105316, -1.4849227666854858, 0.9214842319488525, 0.640487790107727, 0.1767875701189041, 0.20705559849739075, 1.5206332206726074, -1.1679747104644775, -1.313367486000061, 0.6159121990203857, 1.0925250053405762, -0.15898233652114868, -0.2192091941833496, -2.6750924587249756, 0.154892697930336, 1.7274596691131592, -0.4754887521266937, 1.0943852663040161, 2.547213315963745, -0.42670226097106934, -0.8016842603683472, -0.7003398537635803, 1.7922773361206055, 0.19851109385490417, 0.9358656406402588, -0.22832609713077545, 0.9636101722717285, 1.2745205163955688, -0.627722978591919, 0.45542117953300476, -0.32721611857414246, -0.43419209122657776, -0.48464733362197876, -0.7197983264923096, 0.34351643919944763, -0.23869535326957703, 0.5755763053894043, -0.8408645987510681, 1.2302360534667969, -0.6318588852882385, -1.2416820526123047, 0.15047749876976013, -1.6456831693649292, 1.1904579401016235, -2.110621929168701, 2.661062240600586, 0.38583239912986755, 0.9302403330802917, -0.32185012102127075, 0.5634511113166809, 0.07498281449079514, -1.4565330743789673, 0.9830600023269653, -0.37568023800849915, 0.4282259941101074, -0.837456226348877, -0.408686101436615, 0.21771034598350525, 1.1604483127593994, 0.5479445457458496, 0.3642038404941559, 0.6367374062538147, -0.9277457594871521, 1.0565885305404663, -0.5106671452522278, -2.2886500358581543, -2.3533618450164795, -0.605239748954773, -0.0458415225148201, 1.1214518547058105, 0.009482611902058125, -0.4327695071697235, -0.8105243444442749, 1.5737920999526978, -0.39947381615638733, 0.25684669613838196, -1.1675001382827759, -0.7783575654029846, 0.8633459806442261, -0.4860163629055023, 1.8290460109710693, 0.9788554310798645, -0.5129313468933105, 0.705620288848877, -0.2155330926179886, 0.9278833270072937, 0.6737884283065796, 0.8835485577583313, 1.304809808731079, -0.20575441420078278, -0.924831748008728, -0.3128213882446289, 0.17302082479000092, -0.9196088910102844, 0.38225096464157104, -0.48574429750442505, 1.0141061544418335, 0.11280360817909241, 0.14145347476005554, 0.25264254212379456, -0.9775902628898621, 0.5569737553596497, -0.09479398280382156, 0.03321625292301178, -0.004817113280296326, 0.11786246299743652, -2.1158039569854736, -0.7909191846847534, -1.6966729164123535, 0.02007010020315647, -0.9036558866500854, -0.8208579421043396, -0.09101647883653641, -0.9477894306182861, 1.3105264902114868, -0.12510234117507935, -0.3295072317123413, 1.1728981733322144, 1.5808237791061401, 0.3370496928691864, 0.7331644296646118, -0.4201434254646301, 0.4456193447113037, -0.6910632848739624, 0.4665806293487549, 0.9625378251075745, -0.7272314429283142, 0.20342981815338135, 0.9245530366897583, 0.315708190202713, -0.4598599970340729, 1.3262661695480347, -0.09510324150323868, 0.8063445687294006, -1.0968080759048462, -1.2381335496902466, -0.5435270071029663, 0.047011248767375946, 0.8103208541870117, -0.9763001203536987, -0.8079955577850342, 1.309596061706543, 1.0513815879821777, -0.6388832926750183, -0.2819088399410248, -0.4032273292541504, 0.6940776109695435, 0.3619617819786072, -1.1701070070266724, 1.356868028640747, 1.8596277236938477, -0.5710947513580322, -0.7125937938690186, -1.0410346984863281, -0.052267540246248245, -0.9879497289657593, -1.1720764636993408, -2.10455584526062, -0.3697044551372528, 1.0903979539871216, 0.8813216686248779, -0.46954745054244995, -1.1623669862747192, 0.5724307894706726, 0.527739405632019, -2.5783753395080566, 0.5889982581138611, 0.3967348337173462, 0.5986379384994507, -0.4342425465583801, -0.252981036901474, 0.3735934793949127, 0.4691931903362274, 1.1304931640625, -0.20374120771884918, 0.2894953489303589, -0.39015594124794006, -0.5786278247833252, 1.4109716415405273, -1.0989707708358765, -0.985550045967102, 2.1114068031311035, 1.9424192905426025, -2.1537208557128906, 2.3972461223602295, 0.01835796982049942, -0.7656717896461487, 0.334848552942276, 0.41196244955062866, 1.580742359161377, -1.211511254310608, -0.5077081322669983, 0.4474010169506073, -1.8413959741592407, 0.2548988163471222, -2.714824914932251, -0.854897141456604, -0.6031373143196106, 1.1118030548095703, -0.15935514867305756, -0.6085709929466248, 2.582465648651123, 0.5349621176719666, 1.8670926094055176, 0.007941101677715778, 0.2204166203737259, -0.650905966758728, -0.01759198307991028, 0.06175602599978447, 2.872157335281372, 0.3356083035469055, -1.0118554830551147, 0.15502424538135529, 0.03595313802361488, -0.21624600887298584, 0.6975963115692139, 0.15591703355312347, -0.19163143634796143, 0.2480810284614563, -0.9922820925712585, -0.43673229217529297, 1.5028828382492065, -1.058590054512024, -0.553554117679596, -2.3712971210479736, 0.10164419561624527, 0.989401638507843, -1.1669161319732666, -0.20086531341075897, -0.1962471604347229, 1.18755304813385, 0.5719578266143799, 0.6560503840446472, 1.0113474130630493, 1.4665437936782837, 0.41025111079216003, -1.3792695999145508, -0.20651526749134064, -0.06665600836277008, -0.7232932448387146, 0.46978747844696045, -2.9323296546936035, -1.2477387189865112, -1.9208168983459473, 1.1199278831481934, -0.02848459966480732, -0.4790545105934143, -0.5881837010383606, -0.27591708302497864, 0.4839720129966736, -0.6622037291526794, -0.2955392003059387, 1.5942806005477905, 0.5320041179656982, 0.22820809483528137, 1.567865252494812, 0.4470316469669342, -0.28523218631744385, 0.42744603753089905, -0.8387807607650757, 0.6607296466827393, -0.8920167684555054, 0.3696921169757843, -0.29454880952835083, -1.174950122833252, -1.5452693700790405, 0.47545158863067627, -1.8237303495407104, -0.7564837336540222, -0.7184627652168274, -0.03727717325091362, 0.7862943410873413, 0.33144548535346985, 1.2438406944274902, -0.3773336112499237, -1.2519489526748657, 0.23572002351284027, -1.8632668256759644, 0.6865999102592468, 1.8597021102905273, -0.8020803928375244, -0.7077349424362183, -0.8818884491920471, -1.2750486135482788, 0.1161254495382309, -0.2515161633491516, -1.257627248764038, -0.489810973405838, -0.17834119498729706, 0.3296380341053009, -0.32754841446876526, 0.9723283052444458, -0.23321685194969177, -0.5140200853347778, 0.6402468681335449, -0.8174542188644409, 1.9520183801651, -2.630040168762207, 0.9552507400512695, -0.4458630084991455, -1.5946365594863892, 0.18447282910346985, 1.1386592388153076, 0.8770866394042969, 2.42818284034729, -1.260568618774414, -0.69747394323349, 0.5069252252578735, -1.1717017889022827, -1.1770673990249634, -0.8628518581390381, -0.03782173618674278, -0.5918712019920349, -0.46247461438179016, 1.5993640422821045, 0.46128472685813904, 0.31378868222236633, 1.0962474346160889, 1.2795690298080444, -0.04912881925702095, 0.09452848136425018, 0.925345778465271, 1.7691668272018433, 0.9221677780151367, -0.8795368671417236, -0.91890549659729, 2.0918445587158203, -0.07940345257520676, 2.6801681518554688, 0.4563961625099182, 1.4006303548812866, 2.073131799697876, 1.9098771810531616, -0.10821352899074554, -0.33362141251564026, -0.04359393194317818, -0.64475017786026, 1.5654981136322021, -1.4988701343536377, 1.058498501777649, -0.6061468720436096, -0.46230316162109375, -1.586317539215088, -1.3380218744277954, 0.38321653008461, 0.6331044435501099, 0.01319870725274086, -0.24658291041851044, 0.9487847089767456, 0.5325224995613098, -0.41957664489746094, 1.0942167043685913, 0.4750961661338806, 0.3697494864463806, -2.6792399883270264, -0.46666771173477173, -0.2213156819343567, -0.4591677784919739, 0.3410924971103668, -0.043644677847623825, 0.7252633571624756, 1.7015846967697144, 0.2230806201696396, -0.4244323968887329, -0.08935093134641647, -0.4887390434741974, 1.6069892644882202, -2.4770686626434326, -1.2250415086746216, -0.21393050253391266, 1.139271855354309, -0.3827345669269562, 2.0770859718322754, 1.7699339389801025, 0.13571317493915558, -0.4059087336063385, -1.47101891040802, -0.48427489399909973, -0.6247637867927551, -1.3910447359085083, -1.2425674200057983, 0.5231728553771973, -1.795709490776062, 0.4710734188556671, 0.3501916825771332, 0.039485663175582886, 0.3323286175727844, 2.47249698638916, -0.7869296073913574, -0.14528904855251312, 0.553112268447876, 0.5173154473304749, -0.9581470489501953, -0.9528642892837524, -1.6775219440460205, -0.5357760787010193, -0.48525741696357727, 2.1637661457061768, 0.3771559000015259, 0.5490419268608093, 0.03581782802939415, -1.9058412313461304, 0.07723993062973022, 0.7794085741043091, 0.5938995480537415, 0.5437133312225342, 0.3306849002838135, 2.8682496547698975, 0.4731820821762085, 0.34789758920669556, -0.23696836829185486, 0.8002597093582153, -1.3630499839782715, 0.8381643295288086, 1.5246260166168213, -1.5105482339859009, 0.3502413034439087, -0.4870874881744385, -1.2148088216781616, -0.21884240210056305, -0.5598952770233154, -1.4145058393478394, 0.03797692060470581, -0.7337009906768799, 0.08877861499786377, -1.3203626871109009, -1.2646057605743408, 1.5130313634872437, 0.24730461835861206, -1.667152762413025, -0.475585401058197, 1.6475586891174316, 1.493388295173645, -0.8755035400390625, -0.5647472143173218, -0.512654721736908, -1.8708953857421875, 0.13844522833824158, 0.32149815559387207, 0.17801006138324738, -3.0756289958953857, 0.3306429982185364, -0.7956511378288269, -0.9032543897628784, 0.6630144119262695, -1.0372196435928345, 1.4338864088058472, -0.6122223138809204, 0.04126223921775818, 1.3220469951629639, -0.6178367733955383, 0.3018985390663147, 0.2929721474647522, -0.005175379104912281, 0.6402304172515869, -1.1727118492126465, -0.6038376092910767, -0.29252511262893677, 0.4947458803653717, -0.6076132655143738, -0.8292503952980042, 1.5208160877227783, -0.8781690001487732, 0.35168009996414185, 0.7458935379981995, -0.6846779584884644, 1.1002206802368164, -1.1593449115753174, -1.3988698720932007, 0.7788223624229431, -0.1416027843952179, 0.6464080214500427, -0.15637607872486115, 0.9299421906471252, 1.7548478841781616, 0.14016960561275482, -0.810042142868042, 0.20545943081378937, 0.7698556184768677, -0.13362549245357513, 2.6084983348846436, 0.2234504222869873, 0.595180869102478, -0.6787671446800232, -0.04594952613115311, -0.8427859544754028, -1.493383765220642, 1.3220398426055908, -1.3159972429275513, -1.0744558572769165, 0.27467235922813416, -0.5266144275665283, 0.44573912024497986, -0.19021694362163544, -1.3413153886795044, -0.17792312800884247, 0.3997950255870819, -0.4994299113750458, -0.027347728610038757, -1.0691312551498413, 0.08277861773967743, 1.2218339443206787, -0.3155667185783386, 0.4410226047039032, -0.29320257902145386, 0.48189055919647217, -0.5526196360588074, 0.2621175944805145, -0.20239143073558807, -0.030811453238129616, -1.2175917625427246, 1.4273357391357422, 1.1480809450149536, 0.5567622780799866, 1.8070611953735352, -1.1394920349121094, -0.27749380469322205, 0.5660423040390015, 0.12296000868082047, 0.8303928971290588, -0.2021753340959549, 1.3158411979675293, 1.0667033195495605, 2.554715871810913, -0.8426014184951782, -1.5148276090621948, 1.0942076444625854, 0.4881533980369568, -0.39566200971603394, -0.8600323796272278, 0.29978740215301514, 1.513571858406067, -1.846946358680725, -0.5488418340682983, -0.6885294318199158, 0.03616064786911011, 2.0156455039978027, -0.423168808221817, -0.5423556566238403, 0.09997779875993729, 1.0303049087524414, -0.3094206154346466, 1.1592332124710083, 1.249220848083496, 1.5502772331237793, -0.8542933464050293, -0.7655753493309021, -0.17656078934669495, -2.9643983840942383, -1.6812808513641357, -1.3099420070648193, -1.3418197631835938, -0.6408562660217285, 1.4896221160888672, -0.9633373022079468, 0.6521332263946533, 0.3919852674007416, -0.24349510669708252, 2.093775987625122, -0.5015634298324585, -0.29804855585098267, 0.9093930125236511, 0.14927135407924652, -1.6950546503067017, -0.354020893573761, -0.44480276107788086, -1.1177759170532227, 0.3189915418624878, -0.34632304310798645, -0.14092782139778137, 0.5078596472740173, -2.8735830783843994, 0.11479011923074722, -0.08133064210414886, -0.19081376492977142, 1.1319334506988525, 1.7498968839645386, -0.09964194893836975, 0.8493179678916931, -1.1076563596725464, -0.3418804109096527, 1.5589326620101929, -0.5120362043380737, -1.3404937982559204, -0.5874884128570557, 2.1877729892730713, -0.14411544799804688, -2.4352285861968994, 0.18582762777805328, 0.3892260193824768, -0.5687348246574402, 0.40114420652389526, 0.45070457458496094, -0.9796152710914612, 2.5242292881011963, 0.39145272970199585, 0.30519187450408936, 0.8252295851707458, 0.5662530064582825, -0.8797386288642883, 1.3124408721923828, -0.009131391532719135, 0.2898443341255188, -0.7137367725372314, 0.6100345253944397, 1.1823590993881226, 0.37620267271995544, 0.25848180055618286, -1.3905669450759888, -0.5835301876068115, 1.6272207498550415, 0.8006659150123596, 0.45344704389572144, -1.460603952407837, -1.4525716304779053, 1.230212926864624, -0.38468432426452637, -0.8062797784805298, 2.0375888347625732, 0.6687020659446716, 0.6733077764511108, 0.9028903245925903, 0.7659063935279846, 0.7017066478729248, 0.8997489809989929, 0.6358505487442017, 0.7288702726364136, -0.7232218980789185, 0.4545780122280121, 1.520011067390442, -1.4477392435073853, 0.7551641464233398, 0.007272420916706324, 0.12008128315210342, -0.5781126022338867, 0.6443117260932922, 0.6663948893547058, 0.07882165908813477, -0.7454051971435547, 1.3966031074523926, -1.0547939538955688, -0.41083815693855286, -1.2206999063491821, -0.1579492837190628, -0.3718043863773346, 0.07376056164503098, -0.087967649102211, 0.0027703081723302603, -1.4143489599227905, 1.1427531242370605, -1.1140345335006714, 0.5286008715629578, -1.0230207443237305, 0.9604126811027527, -0.08691596239805222, 0.39188268780708313, -0.23608164489269257, 1.1976534128189087, 0.24195654690265656, -0.5440126061439514, 0.5471330285072327, 0.19356149435043335, 0.9108530879020691, 0.31853264570236206, 0.048141349107027054, 0.2551102638244629, 1.4577884674072266, -0.2463730424642563, -0.11360655725002289, -0.8766812086105347, -1.3634692430496216, -0.05296603962779045, 0.25780805945396423, -0.36699944734573364, -0.31128913164138794, -2.0278146266937256, 0.7111445069313049, 1.1187697649002075, -1.3846200704574585, 0.1531832367181778, -0.880516529083252, 0.28753331303596497, -0.8497294187545776, -0.8747833967208862, -1.2016663551330566, 1.535024642944336, -1.7039259672164917, -1.4165174961090088, -0.8627490997314453, 1.6274595260620117, -0.5579697489738464, 0.9479103088378906, -0.991673469543457, -0.06726712733507156, -1.1316711902618408, -0.4036674201488495, -1.721606969833374, -0.05084322765469551], [-0.7510225176811218, 1.0181101560592651, 0.9109607934951782, -1.2851351499557495, 0.030342696234583855, 1.1158709526062012, -0.1356050968170166, 1.1863250732421875, 2.1085002422332764, -1.713206171989441, -0.541249692440033, -1.0558090209960938, -1.3805280923843384, -0.42264577746391296, 0.6205281019210815, 1.4552327394485474, 0.1394699215888977, 0.20391473174095154, -0.8983506560325623, 1.9257453680038452, 1.4782031774520874, -1.239183783531189, -0.3948293924331665, 0.7336616516113281, -0.009184195660054684, -1.7378602027893066, -0.02841583825647831, -2.1334357261657715, 0.1844567060470581, -1.4577076435089111, -0.49234816431999207, 0.163864403963089, -0.80356365442276, -0.1426944136619568, 0.3775022327899933, -0.2754098176956177, -2.137383222579956, 0.9150436520576477, 0.06776260584592819, 0.1938103884458542, 0.24208803474903107, -1.1923279762268066, -0.3733556270599365, -1.1131083965301514, 0.36321642994880676, -0.349391371011734, 0.6833351850509644, -0.19699735939502716, -0.061233289539813995, 0.34596216678619385, 0.8844562768936157, 0.14985939860343933, -1.2334444522857666, 0.26351290941238403, -0.5329300761222839, 0.8575513362884521, 0.7109358310699463, 0.9587486982345581, 1.2437790632247925, -0.24688833951950073, -0.8776430487632751, -0.5517381429672241, -0.3271728754043579, -1.377129316329956, -0.9648926258087158, -0.27260062098503113, 0.847649097442627, 1.159462332725525, -0.5074712038040161, 0.8034250736236572, -1.4996204376220703, -0.7134599089622498, -0.8675501346588135, -0.6703413724899292, 0.025294272229075432, 2.9163825511932373, -0.20075149834156036, 0.027485035359859467, -1.3373535871505737, 0.22894011437892914, 0.7670367360115051, -2.958238363265991, 1.0279172658920288, 0.10985875874757767, 0.6765687465667725, -1.1708893775939941, -1.4126925468444824, -0.5774028897285461, 1.7322646379470825, 0.05587877705693245, -0.9188588857650757, 1.1410995721817017, -0.2817150056362152, 0.3062301576137543, 1.240728497505188, 1.8149213790893555, 0.6312155723571777, -1.7996039390563965, -0.8791777491569519, -1.3062750101089478, -1.207714319229126, -0.40647417306900024, -2.2378695011138916, 0.9475147128105164, 0.8547398447990417, -0.43547385931015015, 1.185088038444519, 0.5290622711181641, -0.8375164866447449, 0.884759783744812, -0.45430123805999756, -0.32916712760925293, -1.0499744415283203, -0.31348124146461487, -1.6880356073379517, -1.1490468978881836, 0.8690329194068909, 0.8283204436302185, -1.498378038406372, -0.8473455905914307, -0.5759401321411133, -2.5218141078948975, -0.3239857852458954, -0.7006992697715759, -0.20887906849384308, 0.12160630524158478, -0.9696463942527771, 1.1207401752471924, 0.36755508184432983, -1.0183024406433105, -0.3172975778579712, -0.1199888214468956, -0.5644400715827942, 0.14389905333518982, -0.3831876814365387, -0.5941007137298584, 1.5812053680419922, 0.3665741980075836, 0.9408949017524719, 1.1572721004486084, 2.0592803955078125, -0.3933473229408264, -0.017547741532325745, 0.28347164392471313, -2.4789016246795654, -0.5895613431930542, 0.28177815675735474, -1.2916278839111328, -0.18802067637443542, -1.7905097007751465, -1.754104733467102, -0.7147706747055054, 0.8185052275657654, -0.008936760947108269, 0.7608121037483215, 0.6145424842834473, -0.7326691746711731, 0.9596357941627502, -1.3053430318832397, 0.7049142122268677, 2.2832798957824707, -1.0345573425292969, -0.1431048959493637, -0.3365841805934906, 1.1887620687484741, -2.7190327644348145, -0.7027767896652222, 0.5983477234840393, 1.592368245124817, -0.911109447479248, 1.406380295753479, -0.4929047226905823, -1.202620506286621, 0.3543558120727539, -1.9612624645233154, -1.4564794301986694, -0.22402025759220123, 0.4919885993003845, -0.8654336929321289, -0.19783082604408264, -0.028156863525509834, 0.6757287979125977, 0.04450279474258423, 1.2552889585494995, -0.6659188270568848, 0.05541829764842987, -0.7045175433158875, -2.1008849143981934, 1.3741167783737183, -0.8221219778060913, 1.571386694908142, -0.8932645320892334, 0.7852214574813843, -0.1227843314409256, -1.0256706476211548, 1.4747284650802612, 0.3445892632007599, 1.1145802736282349, -1.2868105173110962, -1.8293044567108154, -0.1319238692522049, 0.8387174606323242, -0.0809292271733284, 0.8486295938491821, -0.11216045916080475, 0.2245229333639145, -2.2091550827026367, 0.911784291267395, 1.0577659606933594, 0.137876495718956, 0.5588994026184082, 1.020919680595398, -1.2849537134170532, -0.8614353537559509, 1.2486305236816406, -0.12250757217407227, -0.4190944731235504, 0.7000569701194763, -1.8277678489685059, 0.17921602725982666, 0.9466264843940735, -1.5683246850967407, 0.3440890610218048, -0.7609363794326782, 0.20170675218105316, 1.0358498096466064, -0.6747334599494934, 0.5993282794952393, 0.0759567841887474, -0.41945964097976685, 2.0648722648620605, -1.3763736486434937, 1.0294982194900513, 0.9812350869178772, -1.096936583518982, 1.208821177482605, -0.5807958245277405, 0.5236891508102417, -1.1930997371673584, 0.8309802412986755, -1.3380571603775024, -0.20195387303829193, 0.2263047844171524, 0.6282185912132263, 0.9847531318664551, 0.6054165363311768, 1.9969887733459473, 0.5291854739189148, 0.39106065034866333, 0.2243092954158783, 1.7136189937591553, 2.1291728019714355, 0.8852230310440063, -1.7628047466278076, 2.192523717880249, 2.5193357467651367, 1.1803858280181885, -0.24367554485797882, 0.013922459445893764, 0.29797956347465515, -0.04637513682246208, 0.8915020823478699, 0.22319740056991577, 0.4011096656322479, -0.2461843490600586, -1.8209621906280518, -0.2353373020887375, 1.0606776475906372, 0.7208718061447144, 0.8464798331260681, 0.42629873752593994, 0.06756988167762756, -0.6487599015235901, -1.244376540184021, -0.6960196495056152, 0.6174665093421936, 1.6913427114486694, -0.6247517466545105, 1.4974790811538696, 0.6480775475502014, -1.5035881996154785, -0.678527295589447, -1.2119536399841309, -0.24772852659225464, -0.2822909951210022, -0.2713052034378052, 1.7931160926818848, 1.7485346794128418, 0.28336212038993835, 0.08628899604082108, -1.669920563697815, 0.06555244326591492, -0.23285502195358276, -0.6076516509056091, -0.158090278506279, 1.7876719236373901, 0.4958746135234833, -1.1532914638519287, 0.266163170337677, 0.5417388081550598, -0.23957352340221405, 0.5597115755081177, -0.24454852938652039, -0.53397136926651, 0.2537246644496918, -0.17760024964809418, -0.6776988506317139, -1.2677186727523804, 0.2958274185657501, 0.6621925830841064, -0.9721782207489014, -0.3839093744754791, 1.598309874534607, 1.2671600580215454, -0.10786651074886322, 1.4746938943862915, 0.11571469902992249, -0.405875027179718, 1.1830048561096191, -0.10360141098499298, -0.5542193055152893, -1.070886254310608, -0.7639591693878174, 0.46065911650657654, 0.749226450920105, -0.3195846378803253, -0.030839432030916214, 0.6618943214416504, 2.15073561668396, -2.4462270736694336, -2.261531352996826, 1.9227718114852905, -1.3892406225204468, -1.9894635677337646, -2.0174710750579834, -1.7492080926895142, -1.2008103132247925, -0.13711903989315033, 1.9991345405578613, -0.8365187644958496, -0.13346369564533234, 0.21405388414859772, -2.307722806930542, -0.1452697217464447, -2.510040044784546, -0.888724684715271, 1.2655748128890991, -0.6845869421958923, -0.7464091777801514, -0.25397399067878723, -0.4113442599773407, -0.566374659538269, -0.8274223208427429, 0.614508330821991, 2.393967866897583, -0.8580566048622131, 0.30833014845848083, -0.49646735191345215, -0.3449840247631073, -0.4577784538269043, 0.6076893210411072, -2.16363263130188, -1.4430265426635742, 0.4837275743484497, 1.1767717599868774, -1.506911039352417, 0.2682788670063019, -0.010009989142417908, 0.7632147669792175, -0.9699152112007141, 1.439870834350586, 0.15632371604442596, -0.7246590256690979, 1.5137871503829956, 1.637141227722168, -1.3277932405471802, 2.769436836242676, 1.990275263786316, 1.5881729125976562, 0.3201313316822052, -0.8303582072257996, -0.5117862820625305, -0.8667654991149902, -1.725737452507019, -0.6878291964530945, -1.7077280282974243, -1.6684049367904663, 0.4883156418800354, 0.41868019104003906, -0.5764538645744324, -0.413147896528244, -0.23450854420661926, -0.2197428047657013, 1.3001145124435425, -0.3595389425754547, -0.3467619717121124, 0.23800447583198547, -1.1272127628326416, 0.6647087931632996, -0.28416794538497925, -0.566139280796051, 0.07085222005844116, -0.30149203538894653, -0.2396434247493744, 0.5656969547271729, -0.4558951258659363, 0.7918369174003601, -0.5895694494247437, 1.1428675651550293, 0.4768231213092804, 0.3826925754547119, 0.6658347845077515, 1.352556824684143, -0.023744333535432816, -1.6450685262680054, -0.09153088927268982, 0.6359716653823853, -0.8515333533287048, 1.205312967300415, -0.7167476415634155, -0.2716763913631439, -0.5190286040306091, 0.8167337775230408, -0.3649413585662842, 2.308549642562866, -1.1992850303649902, 1.2413439750671387, 1.0443724393844604, -0.9529654383659363, -1.7447757720947266, -0.6241533160209656, 0.3140338957309723, 0.9620461463928223, -0.7464327812194824, -1.0155551433563232, 0.5175732970237732, -0.5062868595123291, -0.7249934077262878, 0.18371789157390594, 0.9915099740028381, 1.1269124746322632, -1.5867729187011719, -0.8298343420028687, 1.2496200799942017, -0.8620142936706543, 0.5502833127975464, -1.1944828033447266, -0.1855422556400299, 1.3478069305419922, 0.4178140163421631, -0.014300555922091007, 0.21909795701503754, 0.689240574836731, 0.05071175470948219, -0.18056944012641907, -0.8613444566726685, -0.55411297082901, -0.019893841817975044, 0.4356180727481842, 0.4168747365474701, 1.0437763929367065, 0.1942024677991867, -0.24312475323677063, -0.719627320766449, 0.1480642408132553, -1.7322824001312256, 0.27533459663391113, -0.5518491268157959, -1.0728368759155273, 1.0160094499588013, 1.0255268812179565, -1.0968672037124634, -2.255751609802246, -1.015468955039978, -0.24309539794921875, -0.5748960375785828, 0.07741311937570572, 1.5311704874038696, -0.5880023837089539, -1.4859260320663452, 1.5047686100006104, 0.8216975927352905, -0.05908301845192909, 0.550883412361145, 0.15179719030857086, -2.0514204502105713, -0.4819926619529724, -0.0801665186882019, 0.35289645195007324, 0.8669800758361816, -0.9574446082115173, 0.28137126564979553, 1.4367977380752563, -1.9384845495224, -0.5027665495872498, -0.06371155381202698, -1.2150737047195435, 1.2666996717453003, 1.140027642250061, -0.5416771173477173, 0.08037778735160828, -2.7389354705810547, -0.8640464544296265, -0.3146703839302063, 0.12582910060882568, -1.2383123636245728, -2.288780689239502, -2.731757402420044, 0.5096610188484192, -0.5586121082305908, 0.1305006742477417, 0.925021231174469, 0.17065425217151642, 0.4048129916191101, -0.42177343368530273, -0.9032566547393799, -0.42537426948547363, -0.11831347644329071, -0.6597905158996582, -0.9574078917503357, 1.5400607585906982, -1.298207402229309, 1.493530035018921, -1.7417869567871094, 0.19101423025131226, 0.5581647157669067, -0.1528187096118927, -1.193482518196106, -0.7588043808937073, -0.2563513219356537, 0.12796930968761444, 1.2174251079559326, 1.3152713775634766, -0.7256630659103394, 0.062490709125995636, -1.1958045959472656, -0.3940245807170868, 0.9213454723358154, 0.34654292464256287, 0.15344412624835968, 1.57164466381073, -0.7157987356185913, -0.5727871656417847, -0.15306274592876434, 0.3168916404247284, 1.0570549964904785, 0.5792443156242371, -1.8402742147445679, -0.18367283046245575, -0.7651137113571167, -1.0844743251800537, 1.3435546159744263, -0.42900022864341736, -0.5008564591407776, -0.772726833820343, -1.7230257987976074, 0.9198799133300781, -1.2549480199813843, -1.0725185871124268, 0.4742780327796936, -0.46571943163871765, 0.08027186989784241, 1.3196024894714355, -1.1137361526489258, 0.9082843661308289, 0.6476775407791138, 0.04418307542800903, -0.5917918682098389, -0.9919440150260925, -1.1248395442962646, -0.051657285541296005, 0.18582090735435486, -1.839266061782837, 1.192980170249939, 0.7293859124183655, -0.36518433690071106, 2.138826847076416, -0.011853275820612907, -1.1341402530670166, -1.8876135349273682, -0.4617367088794708, -0.2730918824672699, -0.016175154596567154, 1.2347713708877563, 1.9799705743789673, -0.9811103343963623, 0.29762011766433716, -0.847004771232605, -0.36847418546676636, 0.3486683666706085, -0.45591872930526733, -0.1771145910024643, 2.185864210128784, -0.8048867583274841, 0.10368689894676208, -0.8621338605880737, 0.8739189505577087, 1.1821120977401733, 0.1616102159023285, 1.4929391145706177, -1.1480544805526733, -0.9568116664886475, -0.3591531813144684, -1.5871275663375854, 0.7980390191078186, 0.23067772388458252, 0.14208362996578217, -0.9635466933250427, 0.6751053333282471, -0.43181928992271423, -0.9483113288879395, -0.1325349509716034, -1.7200967073440552, -0.575960636138916, 1.4573367834091187, -0.062109582126140594, 0.5843828320503235, 0.9375651478767395, -0.23870031535625458, -0.3936016857624054, -1.2463374137878418, -0.6389467716217041, 1.0891451835632324, 1.2576850652694702, -0.3916826844215393, -0.443230539560318, -0.8100345730781555, -1.1036738157272339, -1.1109426021575928, 0.5381490588188171, -0.46131646633148193, -1.5799471139907837, 0.12602713704109192, -2.7181153297424316, 1.614619493484497, 0.3097389340400696, 1.5943447351455688, -1.3857282400131226, 0.6792236566543579, -0.12944673001766205, 1.2695329189300537, 0.18016104400157928, 0.9445257186889648, -0.2167396992444992, -1.4840697050094604, 0.5975584387779236, 0.4114890396595001, 0.6651010513305664, 2.007843017578125, -0.23354369401931763, -1.0223896503448486, 0.219477578997612, 0.324804425239563, -0.02779313735663891, 0.510723352432251, 2.08778715133667, 1.300053358078003, 1.2916059494018555, 0.13863971829414368, -1.4524730443954468, 0.9222527742385864, -0.9328131079673767, 1.7167184352874756, 1.6489218473434448, -2.11669921875, 1.3923118114471436, 0.6887367367744446, -0.28507232666015625, 0.5451503992080688, 1.3786956071853638, -0.7548683881759644, -0.9014193415641785, -1.3939213752746582, 1.340822458267212, -2.04663348197937, -0.7845979928970337, -1.4574007987976074, -0.6690040230751038, -0.8329163789749146, 0.20617225766181946, -0.6112224459648132, 2.720738172531128, 0.05717785656452179, 0.4166797995567322, -0.5165780782699585, -0.8877816200256348, -0.4646589159965515, 0.056077923625707626, 0.008983722887933254, -0.7509787678718567, 0.1960146725177765, -0.6332924962043762, -0.3588767349720001, 0.6148161292076111, 0.785921573638916, -0.8163112998008728, 0.18477988243103027, 0.08441819250583649, -1.2712711095809937, -0.4211345314979553, 1.0323870182037354, -0.5750167369842529, -0.8074120879173279, -0.7963207960128784, -1.362354040145874, -0.8113683462142944, 1.106987714767456, -0.5874958038330078, 0.7337376475334167, -0.7461010217666626, -1.2585866451263428, -0.3260813057422638, 1.663309931755066, -0.4307873845100403, -0.037317126989364624, 1.355002522468567, -1.4923343658447266, -0.5215216279029846, 0.46775683760643005, -0.21525849401950836, 0.48126739263534546, -0.27323293685913086, -1.3925659656524658, -0.7892788648605347, -1.9359078407287598, 1.2996084690093994, -0.9920778870582581, -0.2729089558124542, -0.4869556128978729, -1.3876572847366333, 0.41971611976623535, 1.8534393310546875, -0.26727330684661865, -2.343891143798828, -2.114500045776367, 0.18383601307868958, 0.23052741587162018, -0.17546585202217102, -1.6019669771194458, -1.2693417072296143, 1.682932734489441, 1.1186631917953491, -1.5164004564285278, -1.242775321006775, 0.6171207427978516, 0.9989321231842041, -0.1532769650220871, 0.2502085864543915, 0.5260746479034424, -0.0069759301841259, 0.22390329837799072, -1.6326520442962646, -0.07209424674510956, -1.6378785371780396, 1.0211626291275024, -1.24806809425354, 0.23976336419582367, -0.018377481028437614, -0.3086274564266205, 1.278150200843811, -0.248142272233963, -1.0232058763504028, -1.1244388818740845, 0.6836968064308167, 0.4279051721096039, -0.7487078905105591, 1.6036397218704224, -0.2639228105545044, -0.2548336088657379, -0.3819342851638794, -0.350901335477829, -0.424012154340744, -1.4572962522506714, -1.6458755731582642, -0.7074971795082092, -0.2364896982908249, 0.2029280811548233, -0.5042908191680908, 0.6426191926002502, 0.4362472593784332, -1.2504639625549316, -0.27116698026657104, -0.25519639253616333, 0.05398626998066902, -0.3226149380207062, -0.7881833910942078, 0.3869355618953705, -1.5260483026504517, 3.282383680343628, -1.2388579845428467, 0.39060524106025696, 0.740674614906311, -0.14074884355068207, -1.6703439950942993, 1.1921755075454712, -0.6991723775863647, 0.27789947390556335, 0.523694634437561, -1.0914592742919922, -0.7949646711349487, 0.8318094611167908, 1.390398621559143, -0.7950832843780518, -0.19978933036327362, 1.295287847518921, 0.014869646169245243, 2.6742966175079346, 0.743532121181488, 1.0411741733551025, -0.9171654582023621, -0.8145654201507568, 0.1811656355857849, 0.597851574420929, -1.1232439279556274, -0.16433197259902954, -1.0556517839431763, -0.07248685508966446, -0.11556032299995422, 0.27323514223098755, -0.13262072205543518, -0.6983243823051453, -0.9639225602149963, 1.2909547090530396, -0.09276307374238968, 0.3200981616973877, 1.6186879873275757, 0.02912326715886593, -1.248218059539795, 0.2627333998680115, -1.0280176401138306, 0.5873925089836121, -0.5008191466331482, -0.4288937449455261, 0.3865387439727783, -0.9079155921936035, -2.4100584983825684, -2.1382930278778076, 0.4827754497528076, 0.5486840009689331, 0.6422553062438965, 0.23701754212379456, 0.8909130692481995, 1.4309206008911133, -1.1165190935134888, -0.7913905382156372, 1.241373062133789, 0.5821463465690613, 0.032809674739837646, 0.5136772394180298, 0.4190221428871155, -0.6530942916870117, -0.3364388048648834, 0.9298205971717834, 0.4411664307117462, -0.9332892298698425, 0.4510667622089386, 0.02338937111198902, -0.898589551448822, -1.218793272972107, 0.372378945350647, 0.7148230671882629, -0.29945677518844604, -0.925796627998352, 1.0679352283477783, 0.921124279499054, -0.23536521196365356, -0.6366307735443115, -0.11928635090589523, -1.0171507596969604, 0.04205940663814545, -0.8187106251716614, -0.599583089351654, -1.2205935716629028, -1.3390856981277466, -1.1282830238342285, 1.3400851488113403, -1.0884100198745728, 1.3118566274642944, -1.0334078073501587, -1.38096284866333, 0.15678167343139648, 0.7442668080329895, 0.6087373495101929, 0.8306610584259033, -0.36760562658309937, -0.9926248788833618, -1.0442484617233276, -2.054421901702881, 1.7865222692489624, -0.8840664625167847, 2.2072649002075195, -0.659327507019043, -1.4105303287506104, 0.19484929740428925, -1.0996630191802979, -1.6245172023773193, 0.9354334473609924, -0.33336248993873596, -0.8316102623939514, -0.8957465887069702, -0.6444221138954163, 0.38045257329940796, 0.5042531490325928, -0.4510267674922943, -0.7485299706459045, -0.6029195189476013, -0.3300052285194397, 1.6237084865570068, 1.4057673215866089, 0.4149521589279175, 0.6768608093261719, 1.304907202720642, -1.341984510421753, -1.0250933170318604, -0.5852985382080078, 0.7345356941223145, 0.43339139223098755, 1.0773204565048218, -1.2603851556777954, 0.6820712089538574, -1.3857836723327637, 1.092622995376587, -0.27082669734954834, 0.18879663944244385, 0.3831058442592621, -1.3789803981781006, -0.5010562539100647, 0.7669110894203186, -1.069614291191101, -0.896119236946106, 0.6464930176734924, -0.48415347933769226, -0.29767289757728577, -0.09285938739776611, 0.43547436594963074, 0.016677772626280785, 1.870081901550293, 0.9245257377624512, -0.2705245316028595, -1.1722158193588257, 0.4645368456840515, -0.04124189540743828, -1.4488598108291626, 0.46211254596710205, 0.4641815721988678, -0.3174890875816345, 1.675391674041748, -0.7819257974624634, 1.5184296369552612, -0.12238279730081558, -0.054833974689245224, -0.5427692532539368, 0.726510763168335, 0.1135215237736702, 0.07518384605646133, -0.5874959826469421, 0.049867480993270874, -0.06902552396059036, 0.9425005912780762, 0.14138059318065643, 0.2030155509710312, -0.2828953266143799, -0.17812663316726685, 0.4179360270500183, 0.7484391331672668, -0.5235195159912109, 0.27202093601226807, 0.7203508019447327, 0.8017383813858032, 0.1328316181898117, -0.7940517067909241, 0.547426700592041, -0.13223594427108765, 1.1427711248397827, 0.22575831413269043, 0.37225577235221863, 0.9943197965621948, -1.7309798002243042, 1.034621000289917, 0.6279844641685486, 0.30639734864234924, 2.6684927940368652, 0.8331737518310547, -1.3739020824432373, -0.0035567462909966707, -0.20092879235744476, -0.7267191410064697, -0.5812466740608215, 1.1870813369750977, 0.008733497932553291, 0.814595103263855, -1.1684283018112183, -0.2560155987739563, -0.6736679077148438, 0.07242447882890701, -0.5391491055488586, -0.2674561142921448, 0.36924588680267334, 0.7874046564102173, -1.5056027173995972, 0.016723938286304474, 0.6533704400062561, 0.03463749587535858, 1.6233770847320557, -0.6056337952613831, -1.3699736595153809, -1.2068241834640503, 1.0068411827087402, 0.2388102263212204, -1.1332491636276245, 1.2953925132751465, -0.048093605786561966, 0.42575961351394653, 2.362201452255249, 0.9024233818054199, 2.0250730514526367, 0.37574657797813416, 2.0433003902435303, -1.3652220964431763, -1.6045883893966675, 0.7021316885948181, -0.998799741268158, -0.7879661917686462, -1.9259358644485474, -0.40048426389694214, 1.4370909929275513, -0.41190946102142334, 1.4723331928253174, -0.8806461095809937, -1.7061851024627686, -1.5141270160675049, -0.45602181553840637, -0.07038109749555588, 0.5937548875808716, 0.9904853701591492, -0.10984734445810318, 1.0161893367767334, -0.054399631917476654, 0.3370264768600464, -0.8809322118759155, 0.49048587679862976, 0.9220597147941589, -0.6802463531494141, 0.6356342434883118, -0.26963111758232117, 1.9905755519866943, 0.0238672886043787, 1.2427705526351929, 0.2269650548696518, 0.43513157963752747, 1.1864618062973022, 0.38823744654655457, 1.3203332424163818, 0.09675687551498413, 0.7258456349372864, -0.6839784383773804, -0.3115007281303406, -0.02084389701485634, -0.4397769570350647, 1.0445033311843872, -0.11514946073293686, 0.7352120280265808, 1.889836311340332, 0.7506932616233826, -0.0348573699593544, 0.31695935130119324, 1.6077308654785156, -0.14369021356105804, 1.371738076210022, 1.551171064376831, -1.1355667114257812, 0.6532118320465088, 0.48456454277038574, 0.5944322943687439, -1.0854614973068237, 0.17747223377227783, -0.4629872441291809, 0.2563709020614624, 1.2204993963241577, -0.7348188757896423, -1.0152517557144165, 1.959839105606079, -1.5983954668045044, -0.4756525456905365, 1.2923799753189087, 0.6979451775550842, 1.0380171537399292, -0.2130826860666275, 1.1748366355895996, -2.4451234340667725, -0.08780597150325775, -0.4622412621974945, -0.24806199967861176, -0.3965090215206146, -0.5476177930831909, -1.7749487161636353, -0.6055381894111633, 1.1699941158294678, -0.6512597799301147, 0.4446679949760437, -1.224515676498413, -0.328782856464386, 0.8748835921287537, 0.8668807148933411, -0.17568282783031464, 0.2928023040294647, 0.2622855305671692, -0.8483924865722656, 0.8845762014389038, 0.30872800946235657, 0.8467718958854675, 0.5912963151931763, 1.6606303453445435, 0.4692081809043884, -0.2832258343696594, 0.5922011137008667, 0.5746074914932251, 0.13021211326122284, -0.29359814524650574, -0.4892643988132477, 0.7117961049079895, -0.6072812676429749, -0.5122097730636597, -0.6419717073440552, -1.0665841102600098, -0.3641532361507416, 1.3597337007522583, -0.13817854225635529, -0.6061607003211975, -0.574915885925293, -1.673658847808838, -0.5575479865074158, 0.11394135653972626, -0.6580778360366821, 0.24621427059173584, 0.9264950156211853, 0.466677725315094, -0.6612403392791748, -0.6005329489707947, 0.21506015956401825, 0.9989511370658875, -2.034855842590332, -1.2726961374282837, 1.27881920337677, 0.15092772245407104, -0.7880867123603821, -0.8856518864631653, -0.9217487573623657, 0.8672292828559875, 0.9677212238311768, -0.010143880732357502, 1.6948946714401245, -0.6351138949394226, -0.3264063000679016, -1.1961756944656372, -0.009896119125187397, 0.4754924476146698, 1.8321928977966309, 2.6793766021728516, 0.549858808517456, -0.1903473138809204, -0.622899055480957, -0.012574986554682255, 0.24293051660060883, 0.8071247935295105, 0.2552630305290222, 0.5669516324996948, 0.29272133111953735, 0.1649315059185028, 1.0319898128509521, -0.15144413709640503, -0.984591007232666, -0.7508850693702698, -0.515627384185791, 2.385331392288208, -1.1619694232940674, -0.606865406036377, -0.6338847279548645, 1.9883641004562378, 0.3884458839893341, 1.044506311416626, -1.6457921266555786, -0.20068134367465973, 0.5197350978851318, -0.27848899364471436, 0.9795860052108765, 1.8324254751205444, -1.00185227394104, -0.8627853989601135, 1.110905647277832, 1.2429827451705933, 0.9084308743476868, 1.2924280166625977, -1.2476978302001953, 0.586072564125061, -0.017733827233314514, 0.47497978806495667, 0.4014052152633667, 2.5530600547790527, 2.0794179439544678, 0.18766087293624878, 1.6093038320541382, 1.1207419633865356, 0.8325828313827515, 1.5899933576583862, 1.806423306465149, -0.5917741656303406, -0.6650032997131348, -0.8840543627738953, -0.4952206611633301, -0.30195897817611694, -0.06794935464859009, 0.20836515724658966, -0.436330109834671, 0.19511382281780243, 0.5096317529678345, 1.2845962047576904, 1.0876928567886353, 0.10334302484989166, -0.7894192934036255, 2.391832113265991, -0.7183195352554321, 0.2712882161140442, -0.09608913958072662, -0.2907571792602539, -0.7095407247543335, 0.6990896463394165, -1.7117103338241577, 1.326974868774414, 0.6735457181930542, -0.9810776114463806, -0.19494807720184326, 0.23368360102176666, 0.08796296268701553, 2.647507429122925, 1.0822603702545166, 0.941993772983551, -0.26386547088623047, 0.8540918827056885, 0.08692149817943573, 1.3832619190216064, 0.29871779680252075, 0.20620694756507874, 0.6742753386497498, 2.275521993637085, 0.7533024549484253, -0.3980552554130554, 1.350199818611145, -0.8549244999885559, 1.5596073865890503, 1.967402696609497, -0.5284706354141235, 0.6569111943244934, -1.2115883827209473, 1.5427443981170654, -0.4255877137184143, -0.390636146068573, 0.33609873056411743, 1.6899157762527466, -1.0312572717666626, -0.3720328211784363, 1.3457517623901367, -0.8990145325660706, -0.31073427200317383, -1.0508813858032227, -0.41486456990242004, 2.3738317489624023, 1.7754677534103394, 0.12702025473117828, -0.07269760966300964, 1.9053608179092407, -0.3972639739513397, 0.7884345650672913, 0.49742409586906433, 0.15137384831905365, 0.3183565139770508, 1.804354190826416, -0.52468341588974, 1.2590508460998535, 0.16930171847343445, -1.4628262519836426, 0.17246873676776886, -0.403026282787323, 1.2405775785446167, 0.9730775952339172, -1.241355538368225, -0.9711005091667175, 0.5030746459960938, 1.2732919454574585, -0.19136276841163635, 0.2287176251411438, -0.08607137948274612, 0.26984846591949463, -0.1835513412952423, -0.2892077565193176, -2.3397293090820312, -0.34436383843421936, 0.49412134289741516, -2.1136512756347656, -0.040502503514289856, -0.43113425374031067, 0.27568989992141724, 0.30421262979507446, 1.030258297920227, 0.593113362789154, -0.21041040122509003, 1.0609616041183472, -0.03236395865678787, 1.3372818231582642, -0.8197601437568665, -0.7013795375823975, -1.2807399034500122, -0.7873818874359131, -1.3870896100997925, 0.6299492716789246, -0.053069084882736206, -1.5731127262115479, 0.9841431975364685, 0.9184993505477905, 0.8474146127700806, -1.272079348564148, -0.862316906452179, -1.8606979846954346, -0.6774014234542847, -0.4784916937351227, 0.7139526009559631, 1.7537298202514648, 1.0172843933105469, -0.2902391850948334, -0.15848326683044434, -1.6033192873001099, -0.3662901818752289, 1.0347658395767212, -0.7271136045455933, 1.3278319835662842, 0.2481488585472107, 2.016940116882324, 0.179252028465271, -0.5143209099769592, -0.8814873695373535, -0.3463638722896576, -1.1553382873535156, -1.6244372129440308, -0.5710232257843018, 0.1531827449798584, -0.30904415249824524, -0.24594813585281372, 0.9831775426864624, -0.5775142908096313, -0.07811664789915085, -1.4720122814178467, -0.3177715241909027, -1.000315546989441, -1.1894961595535278, -0.528508186340332, 0.07804591208696365, -1.1293617486953735, 0.7321881055831909, -0.0026885229162871838, 0.610831618309021, 0.5022503137588501, -2.202089309692383, 1.437059760093689, 1.302580714225769, -2.030596971511841, 2.842352867126465, 0.0022518117912113667, -0.4904164671897888, -1.4040498733520508, -0.47295138239860535, 0.25339218974113464, 1.2992637157440186, -0.17767225205898285, 0.7263057231903076, -0.04112496227025986, -2.1680850982666016, -1.8807172775268555, 0.03063349425792694, 0.5429402589797974, 0.6807538866996765, 0.9053082466125488, -0.7690909504890442, 0.7502579689025879, -0.16076837480068207, -0.09707356989383698, 0.9553848505020142, 0.433628648519516, -0.11796420812606812, 1.1388990879058838, 0.7714830040931702, 0.7124053239822388, 0.2456192970275879, -1.0140589475631714, 1.227229118347168, -0.5843939185142517, -0.5326984524726868, 0.7138521671295166, 0.6952438950538635, 1.4187037944793701, 0.0019467816455289721, -0.2399786114692688, 0.5822883248329163, 1.364475131034851, -0.628510057926178, 0.09940475970506668, 1.2700304985046387, -0.14083950221538544, -1.0703965425491333, -0.9706201553344727, 0.5098897218704224, -1.209046483039856, 0.1506393700838089, -0.2524760365486145, 0.45406585931777954, 0.2989937663078308, -0.3625048100948334, -0.02320965938270092, -1.2114925384521484, -0.42651882767677307, 1.928678035736084, -0.012086770497262478, 1.9633668661117554, 0.9735620021820068, 1.767573356628418, -2.343425750732422, 1.4951704740524292, 0.7639105319976807, -1.1745284795761108, 0.9805229306221008, -0.40520769357681274, 2.1840085983276367, 0.1887209564447403, -0.14458335936069489, 0.16601164638996124, 0.186031773686409, 0.21166737377643585, -0.5162975788116455, 0.03665512427687645, 1.1724491119384766, 0.3886512219905853, 0.8992798924446106, 0.36872124671936035, -0.48862627148628235, -2.19984769821167, -0.6897133588790894, -0.8266893625259399, -0.49108821153640747, 0.2227703332901001, 1.9025577306747437, 0.29877039790153503, 1.622765302658081, -0.9569379091262817, -0.11215805262327194, 0.43098214268684387, 1.5918395519256592, 0.08762794733047485, 0.3911801874637604, 0.48965150117874146, 0.03446444869041443, 0.08763347566127777, -0.18719910085201263, -1.0130536556243896, -0.15949100255966187, -0.002679866272956133, 1.0096670389175415, -1.2577842473983765, 0.6840344667434692, 0.5446236729621887, -0.9762011766433716, 0.9319859743118286, 0.039090875536203384, 0.48274940252304077, -1.0226376056671143, -0.7949314713478088, -0.3232925236225128, 1.3561878204345703, -1.0301616191864014, -0.0261004026979208, 0.3856082558631897, 1.921933650970459, -0.5591415166854858, -1.1018973588943481, -2.3157567977905273, -0.08410166203975677, -1.4722836017608643, -0.5400471091270447, -0.9239226579666138, -0.21581466495990753, 0.4698062539100647, 0.45352789759635925, -0.579839289188385, -0.07882118225097656, 0.6441527009010315, -1.826536774635315, 1.0021177530288696, -1.655271291732788, 0.03506823256611824, 0.1930781602859497, -0.6863321661949158, 0.8160093426704407, 0.35816335678100586, -0.3409714698791504, -1.6166882514953613, 0.5328697562217712, -0.2606288492679596, -0.15930016338825226, 0.286026269197464, -1.8135136365890503, -0.46570879220962524, 0.5464898943901062, -0.9203792810440063, 1.1220815181732178, 0.6284471750259399, 0.47578927874565125, 0.2480880469083786, -0.03723675757646561, -0.35502922534942627, 1.0912340879440308, 0.17136330902576447, 1.2252827882766724, -1.7449795007705688, -0.7800350785255432, 1.9190027713775635, -0.6910067200660706, 0.1491069346666336, 1.3105285167694092, 2.6315019130706787, -0.3561748266220093, 1.0255358219146729, -0.1677273064851761, -0.1293518841266632, -0.37756073474884033, 1.5258753299713135, -0.9419063329696655, -2.5285327434539795, 0.42338013648986816, -0.13491356372833252, -0.3614387512207031, -1.023876428604126, 0.3152158260345459, 0.08808835595846176, 0.13926148414611816, -0.07700319588184357, 2.174811601638794, 0.6487683057785034, 1.9970934391021729, 0.02570878528058529, -0.29511916637420654, -0.4836574196815491, 0.5051528811454773, 1.4853155612945557, 0.16074912250041962, 0.50749671459198, -1.7339469194412231, 0.7197499871253967, -0.04679848253726959, 0.505463719367981, 0.6241005659103394, -0.18084992468357086, 0.08907724916934967, 0.6969116926193237, -0.8342272043228149, 0.7941243052482605, -0.31062936782836914, -0.35392725467681885, 0.1994621902704239, 1.0624538660049438, -0.8286995887756348, -0.30990511178970337, 0.2800646424293518, -0.42192766070365906, 1.6281311511993408, 1.4257606267929077, 0.455570250749588, 0.25032803416252136, 2.0714662075042725, 1.2281180620193481, -0.204649418592453, 0.5165471434593201, 0.8619926571846008, -0.81343674659729, -0.16429197788238525, -0.3299207389354706, -1.17625892162323, 0.19624395668506622, -2.193049669265747, 0.15141914784908295, -0.397745281457901, -0.8262479305267334, 0.42212575674057007, -1.2970314025878906, -0.171924889087677, -0.07692272216081619, 1.5168256759643555, -0.5579094886779785, 0.5303970575332642, 1.9146628379821777, 1.0055205821990967, 0.537909209728241, 0.2737544775009155, -0.7479696869850159, -1.7706153392791748, 0.6489766836166382, -1.8729491233825684, 0.3008570671081543, -0.19998425245285034, 0.8006743788719177, -1.0333596467971802, -1.3168479204177856, 0.7369089722633362, -0.28965091705322266, 1.2859126329421997, 0.003345559583976865, -0.7400060296058655, 1.7601046562194824, -0.11035715788602829, -0.19247347116470337, 0.926570475101471, -1.306549310684204, -0.6356645822525024, -0.9039656519889832, -0.22811463475227356, -1.139243483543396, -0.5603100657463074, -1.0682237148284912, 0.6091017127037048, 0.36334553360939026, -0.9588742852210999, 0.4457675516605377, 0.5229776501655579, -0.735984206199646, 0.020643170922994614, 0.2123887538909912, -0.47118228673934937, 0.03728380426764488, -1.731174111366272, -1.7521018981933594, 0.7752130627632141, -1.0858955383300781, 0.4393356740474701, 1.0232300758361816, 0.31178954243659973, -0.40617358684539795, -1.964903473854065, -0.571809709072113, -0.03543525189161301, -0.8185595273971558, 0.4476485848426819, 0.21519654989242554, -0.61175936460495, -0.6174609661102295, -0.0062811486423015594, -0.08720513433218002, -0.47663217782974243, -0.6351936459541321, 1.1519593000411987, 1.6028355360031128, -0.22210539877414703, 0.6923721432685852, -0.12634342908859253, 0.33756759762763977, 1.4017893075942993, 0.21831068396568298, 1.1183292865753174, 0.2936582863330841, -1.4009332656860352, -0.6210370063781738, -1.550184965133667, -0.8483065366744995, 0.8318804502487183, -1.269320011138916, -0.5141233801841736, 0.6230614185333252, 0.7696521282196045, 2.5450615882873535, 0.1651439517736435, -0.04553729668259621, -0.23658597469329834, -0.6512207388877869, 0.1644715964794159, 1.2874025106430054, -1.824134111404419, 0.18485477566719055, 1.4942108392715454, 0.7551851272583008, 1.8225024938583374, -0.40650686621665955, 1.1959683895111084, -1.2088749408721924, 1.3063130378723145, -1.093908429145813, 0.019712328910827637, 0.5035915374755859, 0.6761266589164734, 0.289926141500473, 1.0830416679382324, 0.0018139134626835585, -0.98893803358078, -1.8443453311920166, 1.0032167434692383, -0.21194882690906525, -0.9368727207183838, -0.0629613921046257, 1.7859315872192383, -0.20581823587417603, 0.29236865043640137, 0.8937820792198181, -1.3659838438034058, -1.139826774597168, -0.4086250960826874, -0.4907420873641968, -0.9694504141807556, 0.2891384959220886, 0.06443007290363312, 1.1346641778945923, -1.1394108533859253, 1.2025736570358276, 0.20940503478050232, 1.0951968431472778, 1.190528392791748, 0.6185753345489502, 1.5435150861740112, -1.6755614280700684, -0.9179639220237732, -0.14068356156349182, -0.19528260827064514, 0.30960944294929504, 0.9754517674446106, -0.031836532056331635, 1.2859914302825928, -2.1973977088928223, 0.15536564588546753, -0.03560372814536095, 1.2595610618591309, -1.1102290153503418, -0.08795026689767838, -0.23548753559589386, 0.62498539686203, -0.39546090364456177, -0.411311537027359, 0.7056483626365662, -0.8147410154342651, -0.20957286655902863, 0.4107261002063751, -0.4004566967487335, 0.4739094078540802, 1.9181197881698608, -0.735036313533783, -1.0116386413574219, -0.1537192314863205, 0.6541143655776978, 1.5340595245361328, 0.010468494147062302, 0.6204783916473389, -0.3357575833797455, 0.23552101850509644, 0.8089879751205444, -0.6337829828262329, -1.036520004272461, 1.4447213411331177, 1.3091073036193848, 0.829247236251831, -0.1319265365600586, -1.7911521196365356, -1.4808440208435059, 0.35721269249916077, 0.2585507333278656, -2.198406219482422, 0.3664547801017761, 0.9504085183143616, 0.0828898698091507, 0.8759161233901978, -1.812228798866272, 1.1141005754470825, 0.8643378019332886, -0.21349161863327026, 2.278888702392578, -0.2809794247150421, -1.4692076444625854, 0.36436963081359863, -0.7077409029006958, -0.5425406694412231, -1.2637790441513062, -1.1588387489318848, 1.2182762622833252, -0.341126412153244, 0.7923805713653564, -1.0316851139068604, -0.34106504917144775, -0.4815475642681122, -0.678682267665863, -0.08004581183195114, -0.11083327233791351, -0.6380303502082825, 0.13337551057338715, -0.1311703473329544, -2.4805572032928467, 1.4467236995697021, 2.3664255142211914, 0.43645399808883667, 0.1473006308078766, 2.010030746459961, 0.49238723516464233, 0.32927998900413513, 0.529029369354248, -0.685788631439209, 0.36567315459251404, 0.43333888053894043, -1.1431809663772583, -0.26865389943122864, 1.9025441408157349, 0.35953497886657715, 0.17446939647197723, -0.6474611759185791, 1.9164273738861084, -0.15779611468315125, -0.49943429231643677, 0.5058870911598206, 1.636714220046997, -0.3549947738647461, -0.30150142312049866, 0.4055440127849579, 1.310682773590088, -0.8692566156387329, 0.6683531999588013, 0.8914439678192139, -0.7910194993019104, 0.4526425898075104, 0.07778088748455048, 0.6938973069190979, -1.017656922340393, -0.27979692816734314, 0.07296399772167206, 1.3761589527130127, 0.16457390785217285, 0.43864262104034424, 0.43512776494026184, -0.009455383755266666, -0.07839486747980118, 0.874896228313446, -0.27363675832748413, 2.5273003578186035, -0.04798122122883797, -0.5533485412597656, 0.1147574633359909, -0.6962606310844421, -1.0927674770355225, -0.31680020689964294, 0.08469165116548538, 0.04735672473907471, 2.8215625286102295, -0.06540276855230331, -0.6538842916488647, 2.6181466579437256, -0.3143194615840912, -1.6505709886550903, -0.9679045081138611, 1.2587212324142456, -0.10234023630619049, 1.3514025211334229, -0.48585596680641174, -0.23080964386463165, -0.31774771213531494, -0.3699425458908081, -0.7700693011283875, 1.737344741821289, 1.0890897512435913, 0.9130457639694214, 1.0584367513656616, 0.2860585153102875, -0.2227729856967926, 1.542134404182434, 1.0760947465896606, -0.22646987438201904, 0.5777296423912048, 1.100131869316101, 0.05010442063212395, 2.5675172805786133, 0.1698433756828308, -0.7819744944572449, 0.12067949771881104, 0.19188109040260315, -0.010142521001398563, 0.3013303577899933, 1.1864688396453857, 0.5152434706687927, 0.12733358144760132, 0.1565755307674408, 0.8383540511131287, 1.240652084350586, -0.1097017452120781, -0.8942838907241821, -0.2228454351425171, 1.213573694229126, -1.6170204877853394, -0.6891875863075256, -0.8284426927566528, -1.1620174646377563, -1.409306526184082, -0.8195692300796509, 0.42447686195373535, -0.9068145155906677, 1.0972150564193726, 0.43497079610824585, 1.3353227376937866, 0.8846638798713684, 0.24740353226661682, 0.8754666447639465, 1.75600004196167, -1.4941819906234741, 0.4884709119796753, -1.651056170463562, 1.4374819993972778, 0.37782520055770874, -0.12704679369926453, 1.583039402961731, 0.7390376329421997, 1.1856651306152344, 0.841801106929779, -0.31643155217170715, -0.5092335343360901, -0.36779555678367615, 1.3991769552230835, 0.24124179780483246, 1.333749532699585, -2.2518906593322754, 0.26460978388786316, -0.46048420667648315, -0.6176339983940125, 0.39594095945358276, 0.7966189384460449, -2.127655267715454, -0.15414224565029144, -0.7397916316986084, -1.5467276573181152, 1.2781840562820435, 0.7553749084472656, 0.644950270652771, -2.197397470474243, -1.369921326637268, -0.23697777092456818, 1.1621443033218384, -0.6365165710449219, -1.0224696397781372, -1.0023400783538818, 1.8629961013793945, 0.39475226402282715, -0.6481653451919556, 1.0063430070877075, 1.2384037971496582, -0.9121686220169067, -1.7027392387390137, -0.12210801243782043, -2.3382925987243652, -1.680616855621338, 0.31265920400619507, 0.033947307616472244, 0.7963880896568298, -0.8138408660888672, -0.7629318237304688, 0.7178336977958679, -2.56040358543396, 0.7837125658988953, 0.19807639718055725, -0.3906799852848053, 0.35025930404663086, 0.464099258184433, 0.34120839834213257, 0.6951558589935303, 1.2684658765792847, -1.8318854570388794, -0.4719502627849579, 0.6672861576080322, 0.13933661580085754, -0.6997093558311462, -0.3295762836933136, -0.3523145914077759, 0.0006828596233390272, -1.4881569147109985, 0.9523219466209412, 1.4224294424057007, -0.06363897025585175, -0.003255300922319293, -0.9504985809326172, -1.5763784646987915, 0.26422402262687683, 0.43401995301246643, 0.35387006402015686, 0.16704021394252777, -0.3785792887210846, -1.10649573802948, 0.049504030495882034, 0.7232100963592529, 1.1177116632461548, -1.688612699508667, -0.745236873626709, 0.7168831825256348, -1.2307072877883911, -1.0680928230285645, -0.5588500499725342, -0.7479315400123596, 0.22188183665275574, -0.5872020125389099, 1.7919912338256836, 0.22081266343593597, 0.7987022995948792, 0.9363563656806946, -0.08701594173908234, -0.47072717547416687, 0.6375730037689209, -0.81227046251297, -1.0219204425811768, -0.12524351477622986, 1.0622769594192505, 0.43636026978492737, 0.4842594563961029, -2.8817803859710693, -1.1154545545578003, -0.6529580354690552, -0.6515436768531799, -0.33084380626678467, 1.5071353912353516, 0.39640599489212036, -0.056322552263736725, -0.35163888335227966, -0.6821319460868835, -0.9869042038917542, -0.1783125251531601, -0.31463301181793213, 0.554955005645752, 1.176855444908142, 0.2586430311203003, -2.209043025970459, 0.14902782440185547, -1.1187496185302734, -0.44038474559783936, -0.853705883026123, -0.978360652923584, -0.3835334777832031, 0.30256733298301697, -0.40067145228385925, -0.7316145300865173, 0.7468336820602417, 1.2036080360412598, 1.3181216716766357, 1.8143906593322754, -0.06877821683883667, 1.713184118270874], [-1.2249919176101685, 0.04766969755291939, 2.001751184463501, -0.8293691277503967, -0.39602431654930115, -0.5782742500305176, 0.20602312684059143, -0.6339380741119385, -0.5666217803955078, -1.2164469957351685, -1.824359655380249, 0.07796508818864822, 0.29901230335235596, -0.7939896583557129, -0.6499832272529602, 0.7089111804962158, 0.741032063961029, -0.9496143460273743, -1.2776329517364502, -0.6199474930763245, 1.9226086139678955, -0.727944552898407, 1.808387279510498, -1.108626365661621, -1.14548659324646, -0.2820933163166046, 0.10096196085214615, 0.1612594574689865, -0.8642886281013489, 0.095030777156353, -0.7843745350837708, -0.7690190672874451, 0.9990100860595703, -2.236687660217285, 0.6763015985488892, -0.722114622592926, -0.9638006091117859, 0.762394905090332, 0.4643577039241791, 0.506110668182373, 0.6358754634857178, -0.4424798786640167, 1.0997215509414673, 0.3958257734775543, -0.07762783765792847, 0.2900709807872772, 0.5385562181472778, 1.512041449546814, 1.2115607261657715, -0.8031299710273743, 0.8080497980117798, -0.05936666578054428, 0.22739668190479279, 0.2387160211801529, -0.969378650188446, 0.859641432762146, -1.827362060546875, -0.18931812047958374, 0.04896112531423569, -1.8547853231430054, 0.5698099732398987, 0.3368735909461975, -0.42797574400901794, -0.8601499795913696, 0.6251047849655151, 0.5628910660743713, 0.44308969378471375, 1.2013728618621826, 0.09942543506622314, 0.7929931879043579, -0.3882162570953369, 0.1491306722164154, 1.6505978107452393, 1.3933967351913452, 1.5642006397247314, -0.021075937896966934, 0.9589378833770752, -0.15219464898109436, 2.7115399837493896, -0.9607161283493042, -1.7896709442138672, -0.8582056164741516, -1.9311519861221313, -2.073662757873535, -0.84096759557724, 0.025666216388344765, 1.7971380949020386, -1.424111008644104, 0.47060346603393555, -0.45334547758102417, -0.4022591710090637, 0.21622787415981293, -0.8214383721351624, -1.4231971502304077, -0.613217830657959, -0.539578378200531, -0.0250600166618824, 1.5903117656707764, -0.5931159853935242, -1.6809200048446655, 0.011601700447499752, -1.3178865909576416, -0.8917977809906006, -2.5122170448303223, -0.4985765516757965, 0.6042208671569824, -1.9019691944122314, 2.2361714839935303, -0.7855188846588135, 2.0678741931915283, -0.9549904465675354, 0.3037809729576111, -0.6769499778747559, 0.9418649673461914, 0.4050860106945038, 1.2145905494689941, 0.1737305223941803, -0.154851034283638, -0.05891803279519081, 1.5329220294952393, 0.13913559913635254, 0.5936335921287537, 0.18372739851474762, 0.03126553073525429, -0.4839805066585541, 1.3920177221298218, -1.7154721021652222, -0.6576222777366638, 1.7404471635818481, 0.3959885537624359, -0.5306203365325928, 0.4404851496219635, -1.3577914237976074, -1.0408995151519775, 0.5920922160148621, -1.2576984167099, 0.8465173840522766, 1.520876407623291, -0.25491198897361755, -1.0367106199264526, -1.9562960863113403, 1.41727876663208, 0.8303478956222534, -1.174737572669983, 0.5816766619682312, -1.0641993284225464, 1.7802271842956543, 1.0708584785461426, 0.37992191314697266, 0.007116010878235102, -1.0558971166610718, -0.41801491379737854, 0.4139121174812317, 0.04171289876103401, 0.28569743037223816, -1.2355518341064453, -0.4552847743034363, 0.08121731132268906, 0.6323294639587402, -1.4859434366226196, 0.7810009121894836, -1.610694169998169, 1.917197823524475, -0.4877125918865204, 1.2365046739578247, 0.24775147438049316, 1.3234920501708984, 0.8667667508125305, -1.46083664894104, -1.3498210906982422, 0.5239320993423462, 0.7317604422569275, 0.5124222040176392, -0.09833233803510666, 0.9330750107765198, 0.5751749277114868, 0.8039726614952087, -0.146859273314476, 0.6057584881782532, -1.501789927482605, 0.27183276414871216, -1.1812520027160645, 0.042443372309207916, -0.6576941013336182, -0.49626511335372925, -1.021858811378479, -0.1977275162935257, -0.6708182692527771, 0.9009084701538086, 1.3617701530456543, 0.28157323598861694, -0.82596755027771, 0.7680429816246033, 1.3070768117904663, 1.194727897644043, -0.92080157995224, -0.6136181950569153, 0.6613908410072327, -0.1616266816854477, 0.3443026840686798, -0.7306722402572632, -0.22297389805316925, -0.6125724911689758, 0.2956011891365051, -1.0776264667510986, -1.7952322959899902, 1.4972126483917236, -2.2068095207214355, -0.6217902898788452, 1.8482251167297363, 0.41844642162323, 1.7541954517364502, -0.732533872127533, 0.13720844686031342, -0.6635566353797913, 0.5657392740249634, 1.5534652471542358, -0.8819859623908997, -1.0726666450500488, -0.5955836176872253, -0.5140728950500488, -0.052283477038145065, -0.24219605326652527, -1.3453724384307861, 1.336342215538025, -0.12387995421886444, -1.932490348815918, -1.0061432123184204, -0.3992542028427124, 1.1984261274337769, 0.171312153339386, -1.8271880149841309, 0.3103967607021332, -0.41219788789749146, 2.159524440765381, -1.3172614574432373, -1.1519800424575806, 0.8108152151107788, 2.7165133953094482, 1.4793915748596191, 0.3804810643196106, -0.9117207527160645, -1.8457492589950562, 0.30172789096832275, 0.03464668244123459, 1.4609591960906982, 1.0595842599868774, 0.14797626435756683, -1.683345913887024, -0.6890566349029541, -0.2539433240890503, -0.2556239366531372, 2.065117120742798, 0.23855027556419373, -2.619551420211792, 0.14361052215099335, -0.5103573799133301, 2.399688720703125, -0.8079602122306824, 0.5956852436065674, 0.051661811769008636, 0.022675635293126106, -0.9410796761512756, -0.8670380711555481, -0.07466397434473038, -0.828720211982727, 0.5515111088752747, -0.005811046343296766, -0.23183774948120117, -0.9763578772544861, 0.4011821150779724, -0.887441098690033, -1.506801962852478, 1.0398427248001099, -0.01069325115531683, 0.18360190093517303, 1.9322056770324707, 0.2727471888065338, -0.13542573153972626, -1.6607426404953003, -0.8392645120620728, 0.710409939289093, 1.9605454206466675, -0.09374731779098511, 0.07443467527627945, 0.9807233214378357, -0.3725457787513733, -0.5768198370933533, 1.2004127502441406, -0.4495505690574646, 0.5755919814109802, 0.648172914981842, 0.9970082640647888, 0.24646012485027313, 1.9628417491912842, -0.1323670744895935, -1.436263084411621, 0.48509925603866577, -0.2791788578033447, 0.6600050926208496, 1.0642472505569458, 0.6671350002288818, -0.6808343529701233, -1.4759140014648438, 0.07373198121786118, -1.3181358575820923, -0.960515022277832, 1.6840866804122925, -1.3515923023223877, 0.5936314463615417, 0.06943630427122116, 0.717598557472229, 0.7928764820098877, -1.024272084236145, 0.19864724576473236, -0.2393437922000885, 1.092889428138733, 1.2396223545074463, 1.792060375213623, 0.40714624524116516, -0.26529157161712646, 0.5342760682106018, -1.2408398389816284, -1.321455955505371, 0.4301186203956604, -2.0838141441345215, -0.5692250728607178, 1.5052679777145386, 0.8007210493087769, 2.8382773399353027, 1.7416046857833862, 0.1880231648683548, 0.613301694393158, 2.0065438747406006, -0.17137761414051056, 2.4545998573303223, -1.160388708114624, 1.8117120265960693, 1.0446017980575562, 1.6617974042892456, 1.130516767501831, 0.42287617921829224, -0.0658520758152008, 0.2417447417974472, 0.11237240582704544, 0.4421417713165283, 1.122212529182434, 0.18766233325004578, -0.859110414981842, -0.8928833603858948, -0.751146674156189, -1.3957724571228027, -1.9221296310424805, -1.6387873888015747, 1.4549531936645508, -0.5029043555259705, -0.5097216963768005, 0.6649258732795715, -0.8853263258934021, -0.1298924833536148, 0.8433303833007812, -0.5828283429145813, -1.3468314409255981, 0.1864996999502182, -0.14356979727745056, 0.30763697624206543, 0.5558116436004639, -1.2497280836105347, 2.135915517807007, 0.9269030094146729, -0.3532145917415619, 0.4339141845703125, 1.511002540588379, -0.64045649766922, 0.25103768706321716, 0.009450840763747692, -2.4604973793029785, 0.41513535380363464, 0.8755310773849487, -0.8204046487808228, -1.3609706163406372, -1.2365386486053467, -0.9192551970481873, -0.7676842212677002, -0.640350341796875, 0.03757704049348831, -2.0090136528015137, 2.8591175079345703, -1.245311975479126, -0.3817083239555359, 0.42281585931777954, -1.1036044359207153, -0.18311908841133118, 1.1967350244522095, -0.6579365134239197, -1.3152439594268799, -1.716495394706726, -1.544754147529602, 0.26078829169273376, -0.4679987132549286, 0.5995612740516663, 1.887679100036621, -0.6268709301948547, 0.036303021013736725, -1.187630534172058, 1.5909209251403809, -0.12113460898399353, 0.1842442750930786, 1.1162751913070679, 2.4816665649414062, 2.3598132133483887, -0.9401410222053528, -2.0748672485351562, 0.09928429126739502, -1.027068018913269, 1.4857605695724487, 1.4608495235443115, 0.28968101739883423, 0.054910168051719666, -1.0720268487930298, 0.8884496092796326, 0.4430961012840271, 0.5501528978347778, -0.6810501217842102, -1.8067599534988403, -0.3236459791660309, -1.6158276796340942, -0.06428845226764679, 0.3944224417209625, 0.13185715675354004, 0.8802842497825623, -0.1539057493209839, -1.482269287109375, 0.5095704793930054, 0.42863747477531433, 0.539184033870697, 0.9210456013679504, 0.6516342163085938, 1.6119948625564575, -0.9406065940856934, -1.186348557472229, -1.4516241550445557, 0.5222061276435852, -0.8686396479606628, -1.3644781112670898, 1.3608310222625732, 1.0119982957839966, -1.0611149072647095, -0.23803256452083588, 1.4719268083572388, 1.2275856733322144, 1.2866915464401245, -0.2653904855251312, -1.6007094383239746, 0.03888349235057831, -1.8494566679000854, -0.3775586783885956, -1.191689372062683, -0.32342657446861267, -0.42241331934928894, 0.04040711745619774, 0.5318372249603271, 0.07220128923654556, -0.2842106819152832, 2.2343790531158447, -0.4004809558391571, -0.9118133783340454, 0.6524785161018372, 0.11617261916399002, 1.6573129892349243, -1.7895792722702026, -0.3085641860961914, 0.15711861848831177, -0.43393808603286743, 1.11832594871521, -0.07846076041460037, 0.050037261098623276, -0.6033854484558105, 1.9054536819458008, 0.46165385842323303, -1.0021971464157104, -0.6415675282478333, -0.944654107093811, -0.035809475928545, -0.5078779458999634, 0.33865320682525635, -0.2918723523616791, 0.922818660736084, -0.45259255170822144, -0.10109685361385345, -0.1703868955373764, -0.04799730330705643, -0.6017336249351501, -0.2514122426509857, 1.0650784969329834, -2.037046194076538, 0.23187649250030518, 0.45643502473831177, -0.8038158416748047, -1.9236987829208374, 1.2725532054901123, -1.3065904378890991, 0.1641395390033722, 0.48729774355888367, -0.8595662713050842, 0.9209752678871155, 0.8105665445327759, -0.8320662975311279, 0.45339134335517883, -0.025579731911420822, -0.05060170590877533, 0.8781881928443909, 0.7169530391693115, 0.38031503558158875, 0.7817242741584778, -0.9736284017562866, -1.19976806640625, 0.9853518009185791, 0.05681000277400017, -0.8285950422286987, -0.44603168964385986, -0.603079080581665, 0.41967394948005676, -1.1345469951629639, 0.6891809701919556, -0.1703183948993683, -0.18762394785881042, 0.3871738612651825, 0.967869758605957, 1.3698735237121582, 1.1056355237960815, -0.6982188820838928, -0.5475547909736633, 1.3125858306884766, -0.896014392375946, -0.23120158910751343, -0.8885563015937805, -0.33531856536865234, 0.017304223030805588, -0.8895642161369324, -1.3576414585113525, -0.368015319108963, -1.8256921768188477, -1.9078656435012817, -0.4015429615974426, 1.1376076936721802, 0.9767932891845703, -0.7874934673309326, -0.11546500027179718, 0.009478670544922352, -0.09356904774904251, -0.03224574774503708, -0.7742802500724792, 0.4450450837612152, 0.9786971807479858, 0.7176627516746521, 1.6191097497940063, -1.0944395065307617, -0.45190101861953735, 0.28617122769355774, 0.6716858744621277, 0.3216794431209564, 1.181838870048523, 0.8244684338569641, -0.5719465017318726, -2.025522470474243, 0.8226105570793152, 0.08828221261501312, -0.6022534966468811, -0.9451576471328735, -0.4055277109146118, -0.6523081660270691, -0.44608965516090393, -1.5471528768539429, 1.1106914281845093, 1.0253630876541138, 1.2867298126220703, -1.9210724830627441, -0.6459546685218811, 0.7703842520713806, 0.5884470343589783, -0.29481080174446106, 0.8780657649040222, -0.44443321228027344, 0.39897000789642334, -0.6323673129081726, -0.017814423888921738, -0.010421132668852806, -2.643754005432129, 2.1995468139648438, -0.15790818631649017, -1.956386923789978, 1.2519649267196655, 0.05611091107130051, 1.3828104734420776, -1.5598376989364624, -1.1614540815353394, 0.33209341764450073, -0.9451845288276672, -0.49331963062286377, 0.6267622113227844, 0.21820704638957977, -0.19142840802669525, 0.4334442913532257, -0.21651867032051086, -0.6197080612182617, 1.4337512254714966, -1.421709656715393, 2.51876163482666, 1.07438325881958, -0.39742568135261536, 0.1833229959011078, -0.3211028575897217, -0.549243688583374, 0.892371416091919, -2.632152795791626, 0.34853029251098633, -1.1758959293365479, -0.6860334277153015, 0.8249272108078003, -0.31081467866897583, -0.47308099269866943, -2.9283015727996826, -1.4177510738372803, 0.4949595332145691, -2.070740222930908, -2.002837657928467, -0.4204426407814026, 0.806709885597229, 1.089914083480835, -1.245415449142456, 0.8857042193412781, -0.537764847278595, 0.3986157178878784, -0.9524829387664795, -1.6123689413070679, 0.44793209433555603, -0.24837546050548553, -0.40824463963508606, -0.9387205243110657, 1.9247276782989502, 0.6382701396942139, -0.45193159580230713, 1.7844805717468262, -0.8321800231933594, 0.5711764693260193, -0.33459606766700745, -0.19813226163387299, -0.6766268014907837, 0.15984413027763367, -0.8329009413719177, -0.05934319645166397, -0.001597887952812016, 2.772406578063965, -1.795628547668457, -0.9873610734939575, 1.0608060359954834, -0.2325049787759781, -0.6455231308937073, 0.2744143009185791, 0.24660155177116394, -0.9670501947402954, -0.20388460159301758, 1.0804595947265625, -0.6281699538230896, 0.5733163356781006, -0.24087326228618622, 0.19841071963310242, 0.48533421754837036, -0.30792227387428284, -0.21220803260803223, -0.5119089484214783, 0.3976348340511322, -0.31395089626312256, 0.3463113009929657, 0.3890919089317322, 0.2798464596271515, -0.028025377541780472, 1.1685373783111572, 0.40412765741348267, 0.13785524666309357, -1.293101191520691, -1.4107333421707153, 0.13019026815891266, -1.1956181526184082, 0.7985038757324219, 0.5719001889228821, -0.06650190055370331, -0.05812922865152359, 0.1183064728975296, -0.7533932328224182, 0.16541171073913574, 0.303170382976532, 1.4591373205184937, 0.29727062582969666, 0.3898235261440277, 1.317093014717102, -1.58427894115448, -1.0867908000946045, 0.6730647683143616, 2.026526689529419, 0.6980595588684082, 0.06834491342306137, 0.6732679605484009, -0.6028633117675781, 0.7542318105697632, -0.7372829914093018, -0.5354153513908386, -1.20242440700531, 0.06791190057992935, 0.26289522647857666, 1.1402461528778076, -0.5043951272964478, 0.7792326211929321, -1.1181188821792603, -0.7495423555374146, -0.5610953569412231, 0.5753487348556519, 0.19371674954891205, 0.33061519265174866, -0.47670498490333557, -0.6410262584686279, -0.0014881908427923918, 0.5959123373031616, -0.9095779657363892, -0.9954125881195068, 0.1901165097951889, 1.5225950479507446, -1.6152383089065552, -0.4183713495731354, 1.2646393775939941, -0.54152512550354, 0.2783302366733551, 0.7566201686859131, -1.6715795993804932, -0.022349698469042778, 0.5853450894355774, 1.6598070859909058, 1.278704047203064, 0.4728585183620453, -0.03405804559588432, 0.32212939858436584, 0.9480055570602417, 1.661817193031311, 0.6999777555465698, 0.9102685451507568, -0.8966887593269348, -0.6155247688293457, -1.31476891040802, 2.1784868240356445, -0.07463094592094421, -1.4664268493652344, 0.04301713779568672, -0.4140063226222992, 1.117032527923584, -0.14006473124027252, -0.3930906653404236, 0.03545798361301422, -1.5785149335861206, 0.2571464478969574, -1.833022952079773, 0.1881478875875473, 2.296485662460327, -0.20230580866336823, 1.3630539178848267, 0.027095600962638855, 1.4931695461273193, -0.18242143094539642, -0.26551082730293274, 1.044165015220642, -0.9271849989891052, 0.16562621295452118, -0.6155019998550415, 1.4308409690856934, 2.188049077987671, 0.9726457595825195, 1.3654885292053223, 1.1614147424697876, -0.04792804270982742, -0.5922788977622986, -0.8037099242210388, -1.5502612590789795, 0.5616501569747925, 0.4843006730079651, -0.7298122048377991, -0.022467009723186493, 0.08773882687091827, 0.17588333785533905, 0.08485692739486694, -0.45775771141052246, -1.0510309934616089, -2.1769633293151855, -1.1150377988815308, -0.36866676807403564, 2.200934410095215, -1.6789953708648682, -0.4830934405326843, 1.326407790184021, -1.4523378610610962, 0.3027297258377075, -1.481177806854248, -0.11971870809793472, 0.3726409375667572, -0.6700347065925598, -0.2932496964931488, 0.30871498584747314, -0.3456142246723175, -0.11357582360506058, 0.2381596565246582, -0.49191123247146606, -0.23130178451538086, 0.5493424534797668, -0.9117116332054138, 0.3581564724445343, -1.0817046165466309, 0.19074079394340515, 0.4476749002933502, 0.5393140912055969, 0.4006677567958832, -0.4321846663951874, -0.6663565039634705, -2.3413994312286377, 1.3260554075241089, 0.305785596370697, 0.18870532512664795, -0.5373503565788269, -1.4660210609436035, -0.11068494617938995, 0.737238883972168, 1.5644524097442627, 0.5028095245361328, 1.10801362991333, 0.315820574760437, 0.820835530757904, 1.3839360475540161, 0.9629745483398438, -1.317955493927002, 0.07585672289133072, -0.21574339270591736, -0.9488201141357422, -0.5059080123901367, 0.24441084265708923, -0.37971872091293335, -0.47522246837615967, -0.15556550025939941, 1.0396533012390137, -0.47499898076057434, -1.0900498628616333, 0.7781215310096741, 0.1374049335718155, 0.7213538289070129, -0.01829952746629715, -2.0806400775909424, -0.9287990927696228, 1.1669663190841675, 0.7625182271003723, 1.3215595483779907, 2.441314697265625, -0.08701721578836441, 1.5247117280960083, 0.06068435683846474, -0.4557015001773834, -0.2581483721733093, 0.5120865702629089, 1.838976263999939, -2.03885555267334, 1.2655869722366333, 0.30700477957725525, -1.3072664737701416, -1.038031816482544, 0.46381843090057373, -2.2479615211486816, 1.8582414388656616, 1.137345314025879, 1.47584867477417, -1.1783246994018555, -0.7026042938232422, 0.002692637499421835, 2.1264045238494873, -0.6590715050697327, -0.6649079322814941, 1.5391507148742676, -1.0843015909194946, -0.7160362005233765, -0.29098716378211975, 1.0150001049041748, -0.968043327331543, 0.7660504579544067, -0.5178158283233643, 0.7124687433242798, 0.01279915776103735, -0.39822137355804443, -0.3838350474834442, 1.0023847818374634, -0.4248417913913727, -0.48541730642318726, -0.6466299295425415, 0.2892421782016754, -1.7598154544830322, 0.6143313646316528, -0.3891115188598633, 0.12885935604572296, -1.082218050956726, 0.8062772750854492, -1.1250807046890259, -0.022879287600517273, 0.6652195453643799, 1.1127729415893555, 0.16481159627437592, 0.9822450280189514, 0.7670308351516724, 1.014622688293457, 0.14050617814064026, 0.7599414587020874, -1.5405408143997192, -0.6741723418235779, -0.1395125538110733, 0.47881847620010376, -0.9835744500160217, -0.4659424424171448, 1.2091361284255981, -0.8031336665153503, 0.2646932303905487, -1.2372926473617554, 1.597386360168457, 0.42408302426338196, -1.575583577156067, 0.023136015981435776, 1.2607601881027222, -0.2921309769153595, 0.1192614734172821, 0.4736379384994507, -0.6631162762641907, 0.39953410625457764, 0.38510364294052124, -0.7393125891685486, -0.8977301120758057, 0.5085814595222473, -0.08988204598426819, 1.4979910850524902, -0.7538395524024963, 0.5906145572662354, -1.3166141510009766, -0.516648530960083, 0.04997531697154045, 0.1671244502067566, -0.34446781873703003, 1.0953402519226074, -1.071458339691162, 0.6954699158668518, 0.730124294757843, 1.9057374000549316, -0.04267261549830437, 1.5765817165374756, -1.3112905025482178, -1.2314866781234741, -0.04290135204792023, -1.0546560287475586, -0.20923352241516113, -1.0701754093170166, -1.421416163444519, -1.054337978363037, -1.6959739923477173, -0.017152082175016403, -0.023485751822590828, 1.6954907178878784, 1.6823108196258545, 0.26507487893104553, -1.3250995874404907, -1.7292202711105347, 0.3120613694190979, -0.008004291914403439, 1.2032139301300049, -0.40269771218299866, -0.4720371961593628, 1.2356534004211426, -0.32266363501548767, 0.40007033944129944, -1.1277086734771729, 1.842831015586853, 0.32839369773864746, -1.4946545362472534, 0.23820281028747559, 1.1158615350723267, 0.627118706703186, 0.4624041020870209, 1.027358889579773, -0.5390510559082031, -0.47474241256713867, -1.8681504726409912, 0.10806985199451447, -0.878943145275116, 1.0887583494186401, -0.12353797256946564, 1.091418743133545, 0.08121786266565323, -0.045609135180711746, 0.47538113594055176, 0.931010365486145, 1.013474702835083, 0.5753728747367859, -0.25995999574661255, 0.07584838569164276, 0.10535556823015213, -0.5193701982498169, -0.4447455406188965, -0.7255766987800598, -1.1627683639526367, -0.14562848210334778, 1.4107357263565063, -0.17514164745807648, -0.06260424107313156, -0.8610436916351318, -0.7233545184135437, -0.5826705694198608, 0.7042959332466125, 0.2137327343225479, 1.429854393005371, 0.16470550000667572, 0.7928952574729919, -1.4299732446670532, 0.9957823157310486, -1.2346112728118896, 0.19034133851528168, -1.2057337760925293, -0.9196242094039917, -0.9558268785476685, -0.9411783814430237, 2.409623622894287, -0.07337845116853714, -0.06713917851448059, -0.2489803433418274, -0.5307339429855347, -2.1405892372131348, 0.5847181677818298, 1.9700733423233032, 0.32254189252853394, -0.9519387483596802, -0.8159275650978088, -0.8909358978271484, 0.6351953744888306, -1.131395697593689, -0.463833749294281, 0.034612711519002914, -0.12117473036050797, 1.9217594861984253, -1.9253159761428833, -0.5281226634979248, -1.1824430227279663, -0.5793668627738953, -0.27325448393821716, -0.4516795873641968, 0.26388880610466003, -0.24025869369506836, 1.0350488424301147, -0.6925655007362366, -0.054976582527160645, 1.0142524242401123, -1.017818570137024, -0.19808454811573029, -0.5271962285041809, 0.9817078709602356, -0.3010284900665283, -0.6016979217529297, 1.6396797895431519, -0.7921883463859558, 0.05656401067972183, -2.0149943828582764, 0.3568906784057617, 0.020442890003323555, 3.180574893951416, -0.9517470002174377, 1.231234073638916, 0.06625697016716003, 0.5722766518592834, -0.5045773983001709, 0.9921808838844299, 1.0766865015029907, -1.0135329961776733, -1.9457906484603882, 0.9765787124633789, 0.4407270550727844, -0.15126670897006989, 0.24341759085655212, -1.521443247795105, -1.0400892496109009, 1.2800168991088867, -0.4265166223049164, -1.2308908700942993, -0.8960897326469421, -0.2025931179523468, -0.045571841299533844, 0.42582637071609497, 2.137454032897949, 0.9614372253417969, -0.003476141020655632, 0.018323419615626335, -0.12644629180431366, -1.1210927963256836, -0.22525636851787567, 2.0014774799346924, -0.18494118750095367, -2.1746468544006348, -0.3842640519142151, 0.3795126974582672, 1.2489687204360962, 0.4169173836708069, 0.41596609354019165, 1.289779543876648, 0.13144397735595703, 0.686378538608551, 0.24272362887859344, 0.07146526873111725, -1.3123939037322998, -0.0929550752043724, 0.1864452064037323, -0.2558097243309021, 0.6942404508590698, -0.8652104735374451, -0.43957510590553284, -1.6723676919937134, 1.101690411567688, 0.24108462035655975, 1.6861133575439453, 0.39893192052841187, -0.6557558178901672, 0.49376246333122253, 0.8195918202400208, -0.8678961396217346, 0.9590672850608826, -1.0321998596191406, 1.1272141933441162, -0.796623706817627, 0.07417329400777817, 1.5131548643112183, 0.09448800981044769, 0.6721416711807251, 0.21718493103981018, 0.935591995716095, 1.116093397140503, -0.8165098428726196, -0.6392269134521484, 0.6192283034324646, -1.4802753925323486, -1.167553186416626, -1.5960607528686523, -1.5426864624023438, 0.04083686321973801, 0.9822731614112854, -1.0109752416610718, 1.2805190086364746, 0.3803502321243286, 1.1705026626586914, -0.1451011747121811, 0.8321548104286194, 1.7491085529327393, -0.6087253093719482, -0.644031286239624, -0.7715285420417786, 0.18771184980869293, 0.47437942028045654, 0.07615962624549866, 1.7908233404159546, -0.6057420372962952, 0.8659343719482422, 0.710517406463623, -0.4934792220592499, -1.262002944946289, 1.0228564739227295, 1.5806734561920166, 1.6051268577575684, 0.387470543384552, -0.9139682054519653, -0.22983066737651825, 0.28987106680870056, -2.1149070262908936, -0.31936517357826233, 2.3081119060516357, 0.6277965307235718, 1.139798641204834, -0.789607048034668, -0.10877669602632523, 0.4308520555496216, 0.8191787600517273, 2.1764039993286133, 2.4693894386291504, -0.5906155705451965, 1.3344253301620483, 0.4512730836868286, -0.8915479779243469, 1.2789586782455444, 0.34726735949516296, 1.3243063688278198, 0.02040788345038891, -0.5614619255065918, -0.06762780249118805, 2.1481802463531494, -0.15278497338294983, -0.859917938709259, -0.14554433524608612, 1.0966112613677979, 0.22568058967590332, -1.5184056758880615, -0.09018199145793915, 0.6495553255081177, 0.07790323346853256, 0.7782227396965027, -0.4838015139102936, 0.6751283407211304, 0.5309594869613647, 0.4286024272441864, -0.22261172533035278, 2.189274787902832, -0.40117377042770386, 0.19381503760814667, -0.4630231261253357, -0.13653847575187683, -2.3341095447540283, 0.407993882894516, 0.025061488151550293, 0.45449158549308777, -0.42290055751800537, 0.800495445728302, -0.7973951101303101, -0.5402382612228394, 0.7089859247207642, -1.6200697422027588, -0.9944705367088318, -1.4926517009735107, 0.5473021268844604, -0.5636039972305298, 0.7453650236129761, 0.21997861564159393, 1.0957542657852173, 0.008912376128137112, 0.20721761882305145, -0.7843363881111145, -1.2192659378051758, -0.030769729986786842, -1.15257728099823, 0.1748485416173935, -1.0604220628738403, 0.3023238480091095, 1.003885269165039, -1.2560662031173706, -0.3849122226238251, 0.4188801050186157, -0.139950230717659, -0.7248486876487732, 0.7037465572357178, -0.15040099620819092, 1.2796965837478638, -1.1272470951080322, -1.7365052700042725, -0.24290233850479126, 0.12406156212091446, 0.6836826205253601, 1.050801157951355, 0.9897477626800537, 2.0994184017181396, 0.4430958926677704, -0.013451892882585526, 0.41225770115852356, -0.2087845504283905, 1.326396107673645, -0.22824932634830475, -0.9116002321243286, -0.17230871319770813, -0.6746341586112976, -0.1942836493253708, -1.3342926502227783, 1.0352938175201416, 0.6054702401161194, -0.5192573070526123, 0.07909329235553741, 1.4179913997650146, 0.3590024411678314, 0.22225606441497803, -0.2992016077041626, -1.3137903213500977, -0.44988980889320374, 1.970252513885498, -0.36670830845832825, -0.321908175945282, -0.4430508613586426, 0.6393660306930542, -2.5351614952087402, -0.7610498070716858, -0.9359967708587646, -1.3095123767852783, 0.621338427066803, -0.5442836284637451, 0.36209622025489807, 0.927969753742218, -0.5422994494438171, 0.23142264783382416, 1.0892146825790405, -0.08675901591777802, 0.759802520275116, -1.0609511137008667, -0.3056846857070923, -0.9754127264022827, -0.040557730942964554, -1.697752833366394, -0.1818370223045349, -1.1037774085998535, -0.9415470361709595, 1.558985710144043, -0.4762409031391144, -0.11316247284412384, -0.9375171065330505, -2.0754685401916504, 0.5404500961303711, -0.5836188197135925, 0.04242326691746712, 1.420546531677246, -0.4631034731864929, -0.12150317430496216, 0.2177545577287674, -0.7118896245956421, -0.9520705342292786, -0.5087679028511047, -0.003626093966886401, -1.3007041215896606, -1.2669053077697754, 0.41249212622642517, 0.33297035098075867, -0.43061578273773193, -0.519196629524231, -0.10701441019773483, 0.3870885670185089, 1.171390175819397, -0.25080838799476624, 0.5202125906944275, 0.19266463816165924, -0.034539442509412766, -1.5128966569900513, -0.1651909500360489, 1.2689731121063232, -1.021373987197876, 0.07950855791568756, 1.5332530736923218, -0.005541895516216755, -2.9304914474487305, 1.5164881944656372, -2.4132437705993652, -0.6833522319793701, -0.4323091208934784, -0.8196352124214172, -0.17008914053440094, 0.3820236027240753, 0.1520213633775711, -2.098623752593994, 0.27349594235420227, -0.9466650485992432, 0.6157630085945129, 1.422587275505066, -1.2553907632827759, -0.9387603402137756, -0.5215035676956177, -1.6279057264328003, -0.25875145196914673, -1.4255177974700928, 1.935891032218933, 0.18106040358543396, 2.366147041320801, -1.206151008605957, -1.522793173789978, 0.8709184527397156, -0.1680021733045578, 1.5067024230957031, 1.4460268020629883, 0.7326225638389587, -1.0690867900848389, -0.1490955948829651, 0.5518903732299805, 0.2688932716846466, 0.6263285875320435, 1.3374805450439453, 0.3388252556324005, -2.1060009002685547, -0.5519583821296692, 0.49114683270454407, 0.6406412720680237, 2.603055000305176, 1.6450837850570679, -2.027804374694824, -0.1350950449705124, 0.7351276874542236, 0.5798512697219849, -0.7116826176643372, -0.7556618452072144, 0.061469223350286484, -0.4671380817890167, -1.6879684925079346, 0.3524084687232971, 0.26406168937683105, -0.034460850059986115, -0.42427659034729004, 1.5431116819381714, -0.6756061315536499, 0.6567647457122803, 1.2757829427719116, -0.9075531363487244, -0.7781033515930176, 0.10231733322143555, 0.1258557289838791, 0.7590126395225525, -0.9423830509185791, -0.7737498879432678, -0.6570906639099121, 0.9539700150489807, -0.0889793410897255, -0.05980342999100685, -2.291105031967163, -0.13697031140327454, 0.42206501960754395, -0.22133682668209076, -0.6103435158729553, 1.237865686416626, 0.4391267001628876, 0.7210982441902161, -1.6255501508712769, 1.500771403312683, -1.3637146949768066, 0.5491347312927246, -0.49967020750045776, -1.4919698238372803, -0.8333607912063599, 1.6505588293075562, 1.1747889518737793, -1.3456319570541382, -0.19048351049423218, 0.2304244339466095, -0.20364968478679657, 0.2271736115217209, 1.5180124044418335, -1.699226975440979, 0.25546514987945557, -1.7177492380142212, 0.9958521723747253, -0.4342769682407379, 0.004047399386763573, 0.4732145369052887, -0.5417601466178894, -0.6540279388427734, 0.4037400186061859, 0.10727214813232422, -1.6613998413085938, -0.4504743218421936, 0.22195929288864136, 0.4984531104564667, -0.9637409448623657, 0.6407586336135864, 0.2641713321208954, -0.7380571961402893, -1.8818039894104004, 0.17566101253032684, -0.1194300502538681, 0.23491020500659943, 0.7011508941650391, -0.15629775822162628, 0.2063656747341156, -0.9927791953086853, 0.12488894909620285, -1.1922316551208496, -0.35782164335250854, 0.5663865804672241, -0.31759199500083923, 1.674083948135376, -1.6447770595550537, -0.6357348561286926, -0.3935966193675995, 0.6830506324768066, 1.0818514823913574, 0.5468538403511047, -0.13919107615947723, 1.0996536016464233, -0.8520466089248657, -1.136471152305603, -0.9284489750862122, -0.08031084388494492, 0.0964624434709549, 1.8100873231887817, -1.0151110887527466, 0.06956446915864944, -1.4800935983657837, -0.5741310715675354, 2.3112926483154297, -0.8254308700561523, -0.7919086813926697, -0.25284674763679504, -1.1516108512878418, -1.4057289361953735, -1.3526432514190674, -0.826478898525238, 0.9331426024436951, 1.177341341972351, 1.4626579284667969, 2.457392454147339, -0.9579137563705444, -0.6665329337120056, -0.5741888880729675, -1.6696257591247559, -1.3115005493164062, 0.6423817276954651, -0.4410237967967987, 0.3176631033420563, 1.2189675569534302, -0.5303863883018494, -1.2837226390838623, 0.2521743178367615, -0.3485497236251831, -0.1912841647863388, -1.103189468383789, 0.3895212709903717, -0.523016631603241, 0.49143069982528687, 0.4737013280391693, -0.616467297077179, 0.21352732181549072, -0.7232666015625, 0.6652107238769531, 0.6357932686805725, -0.9884782433509827, -1.0509850978851318, -0.900667130947113, 1.9651378393173218, 0.3978337049484253, -0.9616778492927551, 0.5586327314376831, -1.387559413909912, 0.9854661226272583, 2.040242910385132, 0.12481763958930969, 0.18065573275089264, 1.6859493255615234, -0.5303654074668884, 0.8523752689361572, -0.018753893673419952, 0.27336546778678894, 0.20606042444705963, 0.1809859424829483, 2.26851749420166, 0.6629266738891602, -1.3467226028442383, -2.9269368648529053, 0.5218096375465393, -1.200738787651062, 0.007427995093166828, 0.5498122572898865, -1.1965432167053223, 0.7435072660446167, 0.816461980342865, -0.5757880210876465, 0.935258150100708, -0.6960601210594177, -0.09390731155872345, -0.7030312418937683, 0.5541725754737854, 1.5700440406799316, 0.46747371554374695, 0.42115330696105957, -0.022880902513861656, -2.3024046421051025, 1.416007399559021, 0.698594331741333, -0.9787406921386719, -0.3579671382904053, -0.9105953574180603, -1.9244784116744995, 2.1249358654022217, 0.9403761625289917, -0.7693151235580444, 0.9763690233230591, 1.169634461402893, -0.13210788369178772, 0.17494909465312958, -0.04635423421859741, -1.6215736865997314, -0.3218362331390381, 1.1505330801010132, 0.7091416120529175, -1.8010122776031494, -0.16384020447731018, -1.323241114616394, -1.033838152885437, -0.6047319769859314, 0.31695041060447693, -1.8911155462265015, 1.1266913414001465, -1.0759780406951904, -0.34679120779037476, 0.6201796531677246, -0.542495608329773, -0.8346062898635864, -0.4308291971683502, -1.066465973854065, -0.4645295739173889, -0.2581545114517212, -1.0238759517669678, -0.09264189004898071, 0.4927835464477539, -0.8496495485305786, -0.0011350197019055486, -1.5181419849395752, 0.4971562325954437, 0.4117685854434967, -1.1380337476730347, 0.16903267800807953, -1.2345119714736938, 0.14431440830230713, -1.94947350025177, -0.3080841898918152, 0.26638081669807434, -0.03346718102693558, -1.0618362426757812, -0.9771232008934021, 0.3871946930885315, -0.18067790567874908, -0.45976483821868896, 0.726254940032959, -1.2738240957260132, -1.0189838409423828, 0.941089928150177, -0.10995497554540634, 0.2750290334224701, 0.512596607208252, 0.015619722194969654, -0.15898923575878143, 0.02644144371151924, 0.6938570141792297, -0.48690393567085266, 0.48482534289360046, -0.17531265318393707, 0.9267980456352234, 0.3162635564804077, 0.5078713893890381, -0.6226668953895569, 0.9210673570632935, -1.3636384010314941, -1.0752297639846802, 0.5012899041175842, -0.034702032804489136, 0.5729141235351562, 0.30448734760284424, 0.5224136710166931, -1.872213363647461, 0.04054359719157219, 0.0382763147354126, 0.8010417819023132, -0.9469585418701172, -0.3975825309753418, 0.6292122602462769, 0.9916068911552429, -2.2190887928009033, -0.16783656179904938, 0.4852100610733032, 1.6058127880096436, -0.9097916483879089, 0.3591971695423126, -1.6626039743423462, 2.6063785552978516, -0.5013797879219055, -0.21656303107738495, -0.7494484186172485, -0.4509308636188507, -0.06753727793693542, 0.7157999873161316, 1.4014981985092163, 0.6922104358673096, -0.45201539993286133, 1.0740801095962524, -0.35866063833236694, 0.4839458763599396, 1.4799301624298096, 1.3977982997894287, 1.8311504125595093, -1.0680516958236694, -0.5083414316177368, -0.35866665840148926, -1.461196780204773, -1.4169429540634155, -0.5840035676956177, 0.3122493326663971, 1.7166980504989624, -0.028757881373167038, -0.4336665868759155, -0.42176976799964905, 0.3912878930568695, 1.7239422798156738, -0.6005251407623291, -0.12766218185424805, -0.21359708905220032, 0.5258375406265259, 0.05797277390956879, 0.2592867612838745, 0.33533668518066406, 0.05214110389351845, 0.3003436028957367, -0.8592087030410767, 0.6925455927848816, -0.18039940297603607, 0.6668977737426758, -1.5862523317337036, -0.4698055684566498, 1.6090130805969238, 0.28731897473335266, 0.4434501528739929, -0.4830257296562195, 1.8205950260162354, -0.03660557419061661, 1.1018356084823608, -0.7636159062385559, 1.7055842876434326, 0.24373041093349457, 0.12186379730701447, 0.9479110836982727, -0.6085889935493469, -1.7909284830093384, -0.01010090857744217, 0.8582752346992493, 0.7006927728652954, 0.20906145870685577, -0.2716922163963318, -0.428130179643631, 1.472230315208435, -0.7933458089828491, 0.39865875244140625, 0.6758586168289185, -1.2471867799758911, 1.0326608419418335, 0.8924813270568848, -0.42933645844459534, -0.10538667440414429, 1.0747253894805908, 0.8434137105941772, -1.109330177307129, -0.07224216312170029, -1.6696007251739502, 1.0173355340957642, -0.145969957113266, -0.9755697250366211, 0.7560766339302063, -0.6505510210990906, 1.3621749877929688, 1.4684683084487915, 1.2816494703292847, 1.0704705715179443, -2.6993658542633057, -0.7967544198036194, -2.165226936340332, 0.6904824376106262, -1.6585224866867065, 0.200217604637146, -1.5739082098007202, 0.31913354992866516, -0.9044874310493469, -1.6749827861785889, -1.8887863159179688, -0.4979172646999359, 1.394149661064148, -0.18181495368480682, 1.6626986265182495, -0.4142206907272339, -0.05015135928988457, 0.7341841459274292, 0.2466987818479538, -0.7733607888221741, -0.5649470090866089, 0.4956095218658447, 0.5128311514854431, 0.43874242901802063, 1.0415186882019043, -0.04220329970121384, 0.08870434761047363, 0.498050719499588, -0.3162640333175659, -1.1356793642044067, -0.304246187210083, -0.27051395177841187, 1.1602240800857544, -0.6643752455711365, 0.667070209980011, 1.7090708017349243, 0.3780779242515564, -0.8529177904129028, -0.6111486554145813, -0.5345954298973083, -0.7106096744537354, -1.5297597646713257, 0.11977197974920273, 0.7217920422554016, -0.3334674835205078, -0.15280944108963013, 0.21333830058574677, -0.5324212908744812, -0.681285560131073, 1.1274144649505615, 0.6274906396865845, -1.0499234199523926, -0.6608999371528625, 1.004820704460144, -2.6325061321258545, -2.257668972015381, 0.37404027581214905, 2.0423364639282227, 0.5676604509353638, 1.9781254529953003, 2.698194742202759, 0.7215579748153687, -1.4130644798278809, 0.40864941477775574, -0.907757043838501, -0.01518800389021635, 0.10681973397731781, -0.16630728542804718, -1.9682941436767578, -1.1599454879760742, 1.129586100578308, 0.312927782535553, -0.12380069494247437, 0.503125786781311, 0.14977988600730896, 1.1236320734024048, 0.8799861669540405, -0.5706560015678406, -2.149644613265991, 2.0428411960601807, 0.11863786727190018, 0.6295928955078125, 0.10063721239566803, -1.2084851264953613, -0.3873956799507141, 0.4332273304462433, 0.13411934673786163, -0.3451637923717499, 2.04880690574646, 0.8897242546081543, 0.6239629983901978, 0.2048930674791336, 0.5728143453598022, 0.43445318937301636, -1.0576927661895752, 0.7541161179542542, 0.0672970786690712, -0.6790148019790649, 0.26351043581962585, 1.7760341167449951, -0.29672637581825256, -1.8841826915740967, -0.17221640050411224, 1.3280478715896606, -0.38907328248023987, 0.9146776795387268, -0.47946831583976746, -0.11165104806423187, 1.3235527276992798, 1.0819768905639648, 0.47567620873451233, 0.11284279823303223, -1.6312694549560547, -0.3623901903629303, -0.5241609215736389, -0.18845148384571075, -0.0875624567270279, -0.04217507317662239, -0.5967134833335876, 0.2418597936630249, 0.49759429693222046, 3.6141061782836914, 0.6215206384658813, 0.5658532977104187, -1.0108870267868042, 1.9710890054702759, 0.43620890378952026, 1.8447837829589844, 1.747493028640747, -1.113906979560852, -0.5013047456741333, 0.7599068284034729, -1.6541413068771362, 0.07418333739042282, 1.2236323356628418, -1.645056962966919, 0.48469898104667664, -0.2668178975582123, -1.0206615924835205, -0.16940326988697052, -0.9242838025093079, 1.8680847883224487, 1.1683770418167114, 0.33642181754112244, -0.7563004493713379, -1.2390809059143066, 0.860958993434906, 0.9848495721817017, 0.37469521164894104, 0.6031545996665955, -0.5766825079917908, 0.4368201792240143, -0.8179243803024292, -0.1536881923675537, 1.1429063081741333, -0.2488514930009842, -0.5611597895622253, 1.0267643928527832, -1.0015665292739868, 1.1945067644119263, -1.1099538803100586, 0.3796718120574951, -0.0669608861207962, -0.3161609470844269, 0.7806821465492249, -0.744988739490509, -1.5938364267349243, -0.0676010325551033, -0.6403789520263672, 0.9962221384048462, -0.17610609531402588, -0.6304559111595154, -1.0384845733642578, 0.13619832694530487, 0.3110031187534332, 1.1135809421539307, -0.5978820323944092, -0.2490723580121994, 0.1479039341211319, 1.048670768737793, 2.2124810218811035, -0.1771242767572403, 0.9228827953338623, 0.950739860534668, 0.9696345329284668, -0.11932909488677979, -0.5881582498550415, 0.4724879562854767, 0.20072373747825623, -0.1812465935945511, -0.35611850023269653, -0.10406408458948135, 0.455780565738678, -0.9165645837783813, -0.8560685515403748, 0.17235031723976135, -0.3868699073791504, -0.8395562767982483, 0.1997266560792923, 0.6778505444526672, 0.17999593913555145, 0.7780201435089111, -0.759610116481781, -0.3575533926486969, -0.4071693420410156, 0.8049970269203186, 0.5744566917419434, 0.031102951616048813, 0.0810244157910347, -0.012942376546561718, -1.5159162282943726, -1.1775739192962646, -1.1909184455871582, 0.19764375686645508, -0.3451329171657562, 0.15475888550281525, -0.5595654249191284, -1.4244810342788696, -0.0891149640083313, -0.04471874609589577, -0.7740219831466675, -0.25377732515335083, -0.06460156291723251, -0.9916028380393982, -1.905527949333191, -0.14548243582248688, 0.507725179195404, -1.370559573173523, -1.2909750938415527, -1.4290152788162231, -0.7701549530029297, -0.6281145215034485, -1.5746105909347534, -0.7927306890487671, -0.6193726658821106, -1.1823534965515137, 1.3211594820022583, 0.09488865733146667, 0.12250328809022903, -0.3897567093372345, 0.6233144402503967, -0.7572999596595764, 0.7491388320922852, 0.22832776606082916, 0.3564106822013855, -1.2646839618682861, -0.3048880696296692, 0.42881134152412415, 2.6705379486083984, 1.3512078523635864, -0.7853004932403564, 1.2569888830184937, -0.9630502462387085, -0.9137597680091858, -1.7569564580917358, 0.7816042304039001, -0.13557316362857819, -0.7922186255455017, 1.3656398057937622, -0.23142680525779724, -1.9427235126495361, 0.9455739855766296, -1.6006511449813843, 0.7870756387710571, 0.9104557037353516, -0.09794369339942932, -2.016993284225464, -0.23802463710308075, -1.3319646120071411, -0.3716529309749603, 0.0041182925924658775, -1.4161553382873535, 0.19859953224658966, 0.7830871343612671, 0.40319716930389404, -0.3770098090171814, -2.019364595413208, -0.18263231217861176, 1.0525609254837036, 1.7537585496902466, 0.37487614154815674, -1.2651596069335938, 0.41393205523490906, 0.30022376775741577, -0.02025090530514717, 1.157682180404663, -1.2074958086013794, 0.25126662850379944, -0.6334580779075623, -0.8245939016342163, -0.13863012194633484, 1.2531496286392212, -0.9056347012519836, 2.0316429138183594, -1.422236442565918, -0.40627726912498474, 0.06602911651134491, 0.006426839157938957, -2.007836103439331], [-0.030521098524332047, 1.0212053060531616, -0.22683100402355194, -0.4148256480693817, -1.6256109476089478, 1.4423763751983643, 1.0079423189163208, 1.1151067018508911, 0.9879591464996338, -1.5364711284637451, 0.710034191608429, -1.4983998537063599, 0.10890480130910873, 0.015585909597575665, -0.4299579858779907, 0.315647691488266, 0.4172719717025757, -0.5735806822776794, 0.7256299257278442, 0.8001812696456909, 1.1522200107574463, -0.769452691078186, 1.1551512479782104, -1.2020667791366577, -1.1104776859283447, -1.3026078939437866, -1.5161551237106323, 0.8417650461196899, -0.5530276298522949, 0.3368206322193146, -1.372118353843689, -0.7625504732131958, 0.564672589302063, -1.2207074165344238, -0.7561126351356506, 1.0589220523834229, -0.822548508644104, -1.8237085342407227, -2.4790894985198975, -0.7897507548332214, -0.2603667080402374, -1.1138043403625488, -0.14479966461658478, -1.1027112007141113, 0.3192060589790344, 0.48677030205726624, -1.7996032238006592, 1.0191662311553955, -2.465550661087036, 1.770451545715332, -0.5209856629371643, -1.9534634351730347, 0.6331762075424194, -1.0109835863113403, 0.18249990046024323, 3.232121229171753, 0.10148923844099045, -1.226776361465454, -0.6653803586959839, -0.14266598224639893, 0.960745096206665, 1.516797661781311, -0.7047832608222961, -0.6182727813720703, 0.19103580713272095, 1.7460545301437378, 1.3222335577011108, 1.5553443431854248, 0.4249647855758667, 0.3774429261684418, 1.4285975694656372, -0.8852839469909668, -1.746209740638733, 0.18689487874507904, -0.26235273480415344, -0.03991860896348953, -0.58608078956604, -0.307137131690979, -0.18015554547309875, -1.4586939811706543, -1.4290648698806763, -0.6274639368057251, -1.9971246719360352, -0.9198983907699585, -0.36485719680786133, -0.45105627179145813, -0.7621127367019653, 0.6991452574729919, 1.0796452760696411, 0.3667493760585785, -0.41840240359306335, -0.23347949981689453, -1.9963701963424683, -0.41154107451438904, 1.5963733196258545, 0.5640157461166382, -1.7575794458389282, 1.2085920572280884, 0.39229509234428406, -0.6997494101524353, -1.6273292303085327, -0.5432155728340149, -0.5488293170928955, 0.38808056712150574, 0.451214075088501, 0.4523935914039612, 1.6460751295089722, 0.22575941681861877, 0.006692272610962391, -0.5975322127342224, 1.1578035354614258, 1.1482865810394287, -0.824088454246521, 0.5771250128746033, 0.9463889002799988, 0.10356578230857849, -2.018831968307495, -0.11127569526433945, -1.550655484199524, 0.9592238664627075, 0.2644307315349579, 1.2664039134979248, 0.48381397128105164, -1.649157166481018, -0.14848221838474274, 1.555983066558838, 1.3466652631759644, -0.4647614061832428, -0.23367658257484436, 0.838849663734436, -0.7629636526107788, -0.8734870553016663, -1.9705314636230469, 0.04349788650870323, -0.054035380482673645, 1.4487392902374268, 0.2779187262058258, -0.26138460636138916, -0.552295982837677, -0.6139653325080872, 0.38486120104789734, -1.7874661684036255, -1.7286064624786377, 1.3159701824188232, -3.287243366241455, 0.36764711141586304, 1.219451665878296, -1.8878822326660156, -1.6079931259155273, 1.8710827827453613, 0.06902837753295898, 0.33197227120399475, 0.31696879863739014, 0.4505611062049866, -0.13556331396102905, 1.8833917379379272, -1.1370782852172852, 0.21201573312282562, 1.6665382385253906, -1.182076334953308, -0.9551204442977905, -0.7266836166381836, 0.22300438582897186, 0.6582808494567871, -0.9634290337562561, 0.25564301013946533, 1.7655993700027466, 0.9577342867851257, -0.5579193830490112, 0.19716967642307281, 0.5269726514816284, -1.2466226816177368, -0.18778860569000244, 0.9529024958610535, 0.7759749889373779, 0.7593887448310852, -0.5039620399475098, -0.007843833416700363, -1.9971232414245605, 1.1462663412094116, -0.6384127140045166, 0.041333939880132675, 0.31962576508522034, -2.3353943824768066, 0.9066802859306335, -0.36580371856689453, 0.6997794508934021, -1.5456571578979492, 0.8984694480895996, 0.8660004734992981, 0.19983921945095062, 0.4113823175430298, -0.228988915681839, -0.18202102184295654, -0.35415205359458923, 1.0766277313232422, -1.0146654844284058, -1.526249647140503, -0.8545148968696594, 0.23459668457508087, -0.02368720807135105, -0.03129536658525467, 2.723411798477173, 0.7415033578872681, -1.5130916833877563, 1.0537703037261963, 1.781409740447998, -0.3812984228134155, 1.758935809135437, 1.683455467224121, 1.332789659500122, -0.8719802498817444, 0.5780084133148193, -1.1105377674102783, -1.358138918876648, -1.0501383543014526, -0.4735916256904602, 0.10101952403783798, 1.3803894519805908, 0.9933540225028992, 0.12343766540288925, -0.7666155099868774, -0.6090489029884338, -0.5489147305488586, 0.6954812407493591, -1.1456656455993652, -1.4741907119750977, 1.6022595167160034, 0.6180122494697571, -0.23258064687252045, -0.7296985983848572, 1.5055381059646606, -0.3578404188156128, -0.25356414914131165, 0.9377734661102295, 0.40912753343582153, 0.987190306186676, -1.8423302173614502, 0.8747686147689819, 0.8842623233795166, -0.6677291989326477, 1.6608375310897827, -0.258117139339447, 0.34273695945739746, -0.6852787733078003, -0.9664922952651978, -0.2893291115760803, -1.3433548212051392, -0.7580646872520447, -1.9669476747512817, -1.1857547760009766, -0.14404645562171936, 0.11506625264883041, -0.6191213726997375, -0.3076091706752777, 0.21564602851867676, -0.8188018202781677, -2.0191941261291504, 0.6669747829437256, -0.22876912355422974, 0.13735705614089966, -2.2232797145843506, 0.06577510386705399, 1.0915162563323975, 0.9493408799171448, -0.04586897790431976, -1.1126497983932495, -1.7261062860488892, -0.3738386034965515, -0.5411086082458496, 1.0506137609481812, 0.5462589263916016, 0.019047940149903297, 0.6085790395736694, 0.5088608264923096, 0.4622231721878052, 0.10482173413038254, 0.5414595007896423, 0.24683500826358795, 0.5384330153465271, 0.624125599861145, -1.081756591796875, -0.09270195662975311, 1.1502158641815186, -0.5105173587799072, 0.27304747700691223, 0.9609575271606445, -0.4335614740848541, -0.2045176476240158, 0.4672013521194458, -1.7225914001464844, -0.5852299332618713, -0.5672168135643005, 1.0761977434158325, 0.6574881076812744, 1.0713979005813599, -0.23170235753059387, 0.40781283378601074, 0.30677685141563416, -0.41849613189697266, 0.6651051640510559, 0.12068784981966019, -0.7008928656578064, 0.6309277415275574, -0.4800611734390259, 2.1228439807891846, -0.4267549216747284, 2.0230422019958496, 0.3977680504322052, -0.41441231966018677, -1.1383051872253418, -1.2803726196289062, 0.3063456416130066, 2.5415022373199463, -0.5837676525115967, -0.49629682302474976, -1.8170323371887207, -1.1287329196929932, -0.4478938579559326, -0.2684461176395416, -0.47508862614631653, -0.8768162131309509, -0.2749407887458801, 0.9336199760437012, 0.0317949578166008, -2.274029016494751, -0.39149099588394165, -0.5457112789154053, 0.11196692287921906, 1.1414237022399902, 0.7164681553840637, -2.0395665168762207, -0.6131583452224731, 0.16807441413402557, 0.012543912045657635, 0.934779703617096, -0.1583501696586609, -0.6934333443641663, -0.6066027283668518, -0.8536697030067444, 0.5199553370475769, 2.0822057723999023, 0.11549067497253418, 0.1670895516872406, -1.2139403820037842, 0.3941648006439209, 0.017378469929099083, 0.3153166174888611, -0.2842511832714081, 1.0495855808258057, 1.5570238828659058, 0.5110066533088684, -0.5324879288673401, -1.5067355632781982, -1.720825433731079, 0.14123360812664032, 1.391797661781311, -0.2561238408088684, -1.2193984985351562, 0.6640937924385071, -0.5400543808937073, 0.79617840051651, 1.0396366119384766, 0.8346712589263916, 1.4986426830291748, -0.26386579871177673, -0.5176109075546265, -0.5242279171943665, -1.3981812000274658, -0.846019446849823, -0.18743661046028137, -0.2304544895887375, -1.170562744140625, 0.9506887197494507, -1.3335391283035278, -0.40267232060432434, -1.5835751295089722, -0.12705181539058685, -1.234757423400879, -1.3433009386062622, 1.4675501585006714, 0.6846717596054077, -0.3691103756427765, -0.5282354950904846, 0.8493269085884094, -0.37783268094062805, -1.4817862510681152, 1.0669593811035156, 1.152802586555481, -0.11920411884784698, 0.2923164963722229, 1.1207767724990845, 0.25825220346450806, 0.43392065167427063, 1.172766923904419, 0.351044237613678, 1.3000870943069458, -0.6823179721832275, -0.4250079393386841, -1.5894556045532227, 0.7888755202293396, 1.0107232332229614, -0.5965169072151184, -0.08354781568050385, -0.07144924998283386, 1.1575860977172852, -1.1172404289245605, -0.6296874284744263, -1.4961802959442139, -1.072150707244873, -0.5671065449714661, 0.29805007576942444, -1.2227245569229126, -0.2615460157394409, 0.8086186051368713, -1.3343967199325562, 0.21396754682064056, -0.6445989608764648, 0.7753292322158813, -0.5120741724967957, 2.1365439891815186, -0.3529297113418579, -0.7088097333908081, 1.2915092706680298, -0.28546106815338135, 0.8617252111434937, 0.17655500769615173, -0.4406597316265106, -0.4349796175956726, 0.7472797632217407, 1.892004132270813, -0.44069087505340576, 0.5964069366455078, 0.8887642621994019, -0.4295007586479187, 1.492773413658142, 0.06254186481237411, 0.3015848398208618, 0.24238285422325134, -0.15505856275558472, 0.3417435586452484, -0.754281222820282, 1.2955797910690308, -0.24939444661140442, -0.17989042401313782, -0.9349506497383118, 1.8273602724075317, 1.1135841608047485, -0.4614662528038025, -1.1802527904510498, 0.12689951062202454, 0.4739852249622345, 0.9650487303733826, -0.7513999938964844, -0.7536360025405884, 0.0064650713466107845, 1.2457013130187988, 1.8475956916809082, -0.3918108344078064, 0.11285334825515747, -0.7709988355636597, 0.0569835901260376, -0.12421251833438873, 0.5804094672203064, -0.01489704754203558, -0.9233325123786926, -0.6064589619636536, 0.9787546396255493, 0.6632150411605835, 1.8093509674072266, 0.4477938115596771, 0.18231196701526642, 0.9508205652236938, 1.311745285987854, 0.46663957834243774, 0.27298226952552795, 0.9730145335197449, -1.682531714439392, 0.8958925008773804, -0.5149558186531067, 1.9430111646652222, -1.0673720836639404, -0.14148907363414764, -0.685710608959198, -1.524505853652954, -0.2602913975715637, -0.4490355849266052, -1.69630765914917, 0.026993731036782265, -2.3377268314361572, 0.6392677426338196, -0.08567162603139877, 0.17950324714183807, -1.024482250213623, 0.2794078290462494, -1.160163164138794, -0.6100947260856628, -0.3854087293148041, 0.7331620454788208, -1.2917166948318481, -1.7565810680389404, 0.634118914604187, -0.5456435680389404, -0.3587092459201813, 0.8674637675285339, 0.010877415537834167, -0.6335477828979492, 0.266038715839386, -1.8705271482467651, 0.4332577884197235, -1.0945051908493042, -0.4878547191619873, 0.44794899225234985, -0.35818779468536377, -1.311067819595337, -0.5245146155357361, 0.8525454998016357, -0.036094971001148224, 0.4391091763973236, -0.8718701601028442, -0.8730820417404175, 0.0539238415658474, 0.3563690483570099, 0.32801494002342224, 0.003967054188251495, -0.5712357759475708, 0.14963877201080322, 1.6370158195495605, -0.1628004014492035, 0.07177697867155075, 0.3725455701351166, 1.500022292137146, 0.8438931107521057, 1.1941992044448853, -0.5410985946655273, 0.9434821009635925, -0.630250096321106, 1.123500943183899, -0.319195419549942, -0.9100181460380554, 0.2471664845943451, 1.4972838163375854, 0.7490829825401306, 0.5831611156463623, -0.43490856885910034, -0.7466151714324951, 0.7974112033843994, -0.4293721914291382, -0.8383746147155762, 1.3653342723846436, 1.3824886083602905, -0.1662740409374237, 0.41735804080963135, -0.14688090980052948, 0.5557144284248352, 0.1557111293077469, 1.9977214336395264, 0.12233889847993851, 1.1393816471099854, -1.8071008920669556, 0.18719159066677094, 1.6574002504348755, -0.5686954259872437, 0.39832744002342224, -0.8480170965194702, -1.261828064918518, 0.5741097331047058, -0.8259247541427612, 0.6853682994842529, -0.387692928314209, 0.5739002227783203, 1.150362491607666, -0.5253821611404419, -1.792811393737793, 0.0385562963783741, 0.021411964669823647, -0.33986979722976685, -1.2348501682281494, 0.09384644031524658, 1.9449316263198853, -1.1602429151535034, 1.1085996627807617, 0.8065809011459351, 0.27971717715263367, 1.1715364456176758, 0.062326546758413315, 1.2715091705322266, 1.16712486743927, -0.5334208011627197, 1.0250364542007446, -0.9256720542907715, -3.161242961883545, 0.575221598148346, 1.295433521270752, -0.41330602765083313, -0.7074452638626099, 0.3060291111469269, 1.0705338716506958, 0.04034245386719704, -0.5805698037147522, -0.32949385046958923, -1.095338225364685, -0.28624969720840454, 0.7490991950035095, 0.6063584685325623, 0.3192269504070282, -0.18218033015727997, 0.7730562686920166, -0.8501250743865967, 0.6990281343460083, 0.5608293414115906, -0.5115551948547363, -0.890796959400177, 0.12235143780708313, -1.6216709613800049, -0.42671945691108704, -2.3265879154205322, -1.0669772624969482, -1.2705219984054565, -0.021889492869377136, 1.6863658428192139, 0.13177640736103058, 0.3591603636741638, 0.9056916832923889, 0.6585007309913635, 0.8866316080093384, -1.5661298036575317, 2.0644562244415283, -1.0732316970825195, 1.7495218515396118, -0.6030001044273376, 0.3092717230319977, 0.6744112372398376, -0.2859016954898834, -0.3206397593021393, 0.8998891115188599, 2.349919319152832, -0.19942611455917358, -0.7173841595649719, 0.9947245717048645, 0.7612913846969604, -0.6260716915130615, 1.3510174751281738, 0.6162991523742676, 0.2135356217622757, 0.4652400612831116, -0.6342365145683289, 1.7095544338226318, 0.10461732000112534, 0.5690847039222717, -0.06130135804414749, 0.841216504573822, 0.22786349058151245, -0.5987341403961182, 0.06469804048538208, -1.1683412790298462, 0.578266441822052, 0.5736790299415588, 0.07402820140123367, -1.02053701877594, -1.0767974853515625, -0.7054136991500854, 0.49682292342185974, 0.7707594037055969, 0.3426342308521271, -0.25887688994407654, -0.7005671858787537, -0.826709508895874, 0.8065922856330872, -0.032204095274209976, 0.6842008233070374, -0.5350579619407654, 0.7763042449951172, -1.1847891807556152, -0.8096861839294434, -0.03298919275403023, 0.36660757660865784, -1.5473233461380005, 0.4735986590385437, -2.7930924892425537, -2.2402193546295166, 0.04509308561682701, 0.28710365295410156, -0.2323618233203888, -1.0293633937835693, 0.789854109287262, 0.41432058811187744, 2.2829954624176025, -1.140164852142334, -0.02953079342842102, -0.3719548285007477, -0.4909546971321106, -1.3809363842010498, 1.4739875793457031, 0.16450758278369904, -1.9821449518203735, 1.2493237257003784, -0.23816102743148804, 0.543024480342865, -1.8464415073394775, 0.008195909671485424, -0.844746470451355, 1.12004554271698, 0.6272494792938232, -0.234526589512825, 0.3198465406894684, 0.5174264907836914, -0.6450098752975464, -0.17848502099514008, 1.393553376197815, -0.7225659489631653, -0.6457756161689758, 1.5714848041534424, -0.1857471615076065, -0.09237227588891983, 0.745063841342926, -0.5459843873977661, 0.1157212033867836, -0.9046323299407959, -0.6562252640724182, -0.7901209592819214, 0.27811506390571594, -0.8831764459609985, -1.0515471696853638, -1.1356525421142578, 1.5941041707992554, -0.6934888362884521, -0.4655786454677582, 0.48655518889427185, -0.37342649698257446, 1.2712578773498535, -0.17892618477344513, -0.013119213283061981, -0.4935033619403839, 0.6428540349006653, -0.6876406073570251, -0.5803877711296082, -0.5752484798431396, -0.1677255928516388, -1.1047173738479614, -0.4593063294887543, 0.06319261342287064, -1.1349105834960938, 0.04218880459666252, -0.8804531693458557, 0.6015708446502686, 1.218833327293396, 1.6036983728408813, 0.07385576516389847, 0.19775645434856415, -1.0966362953186035, 0.438315212726593, -0.30831634998321533, 0.7581430673599243, -0.6725514531135559, 0.0858306884765625, -0.4910557270050049, -0.16352251172065735, 1.0844504833221436, -1.497864007949829, 0.44530877470970154, 0.39688652753829956, -0.08558140695095062, -0.08728887885808945, -1.3840936422348022, -0.3357977569103241, 0.13229712843894958, 0.8608151078224182, -0.02902420051395893, -1.6320940256118774, -0.2539691925048828, 1.1219693422317505, -1.7850806713104248, 0.679213285446167, -1.0407990217208862, -0.31449034810066223, -0.25706931948661804, 0.22689643502235413, 0.033602338284254074, 1.5293364524841309, 1.567847728729248, -1.2742811441421509, 0.1401924043893814, 0.10455455631017685, 0.2277761548757553, 0.8765034675598145, -0.7975500822067261, 0.8686410784721375, -0.13508260250091553, -0.11059008538722992, 1.4747123718261719, -0.5642161965370178, 0.7764027118682861, -1.3630337715148926, -0.9646158218383789, 0.4756524860858917, 1.710503339767456, 0.38838648796081543, 0.20440496504306793, -1.0157657861709595, 0.4288063049316406, -0.037408310920000076, 0.4395240247249603, 0.20291711390018463, 0.4659774601459503, 0.07772500813007355, -1.057610273361206, -0.34318849444389343, -0.6040025949478149, -1.0452518463134766, 1.0014342069625854, -1.89432954788208, -0.6228193640708923, -0.25706350803375244, -0.6061246395111084, 0.26281386613845825, -0.03072461113333702, -0.9275168776512146, 0.7941035628318787, 0.11837274581193924, -0.8267045021057129, 0.22294561564922333, -0.20116591453552246, -0.2776036560535431, 1.291082501411438, -0.19165562093257904, -1.9154866933822632, 0.6803097128868103, -0.5213693380355835, 0.5940861105918884, 0.6235044002532959, 0.20107561349868774, 2.3171346187591553, -0.07355771958827972, -0.15350346267223358, -1.0068652629852295, -2.5651698112487793, -0.4409545361995697, 1.1260511875152588, -0.7101954221725464, 0.7858256101608276, 1.0047088861465454, -0.5345481634140015, 0.8978268504142761, -0.18381497263908386, -1.4060108661651611, -2.054689884185791, 1.3801217079162598, 1.0409908294677734, -2.5728280544281006, -1.0386861562728882, 1.7787829637527466, -1.025794506072998, 0.13628864288330078, -0.5992363691329956, -0.23936451971530914, 0.6487370729446411, 0.6081533432006836, -1.6589237451553345, 1.6061346530914307, -0.19135363399982452, -1.2112394571304321, -0.06877057999372482, 0.4604736268520355, 0.9999949932098389, -0.38353171944618225, 0.12932910025119781, -0.3215893507003784, -1.0460175275802612, 2.702911376953125, -0.02589455433189869, 0.49104636907577515, -1.5746883153915405, 0.748206615447998, 1.10325026512146, -0.21807970106601715, -2.410991668701172, 1.7895750999450684, 0.045802921056747437, -1.3223472833633423, 0.5830283761024475, -1.1560931205749512, 0.8609252572059631, -1.5267202854156494, 0.5761129260063171, -1.8542457818984985, -0.1643718183040619, -0.7336536049842834, -0.7917640805244446, -1.8737996816635132, 0.48824596405029297, 1.3239609003067017, -0.05083533376455307, -0.8271452784538269, -1.5755410194396973, -0.49658697843551636, -0.14123007655143738, -0.9495717287063599, 0.9317911863327026, 0.2179650366306305, -0.6219127178192139, -0.11897104233503342, 1.738478183746338, 0.36159569025039673, 1.2701205015182495, 1.08250892162323, -0.5189584493637085, 1.8774702548980713, -0.8369357585906982, 1.2624787092208862, 1.3381744623184204, -0.2198926955461502, 0.20405977964401245, -1.0288763046264648, 0.8224961161613464, 1.234951376914978, 0.3305939733982086, -0.9821181297302246, -0.2282620370388031, 0.7225774526596069, 0.31290578842163086, 1.0303579568862915, -0.6980651021003723, 0.21891477704048157, 0.6795806884765625, -0.917290449142456, 0.16343243420124054, 0.2023816555738449, 0.30037403106689453, -0.6413126587867737, 0.4151955246925354, -1.1483114957809448, -0.10819991677999496, 1.7373533248901367, -1.8246192932128906, 1.1174280643463135, 1.2848808765411377, -0.07057656347751617, -2.704486846923828, -1.4248589277267456, 0.18466421961784363, -1.526249647140503, -0.12008368223905563, -0.27234989404678345, -1.8086330890655518, -0.48080846667289734, 0.15465153753757477, 0.9149554371833801, 1.9266527891159058, -1.1870720386505127, -0.2125045359134674, -0.21097737550735474, 0.16858398914337158, 1.3251618146896362, 1.473060131072998, -0.9227109551429749, 0.2887533903121948, -1.6647969484329224, 0.5447378158569336, -0.4547218680381775, -1.7555744647979736, -0.14965705573558807, -0.08479330688714981, -0.9440858364105225, 0.6872025728225708, 0.43251892924308777, 0.883086085319519, -0.6618712544441223, 0.8201170563697815, 0.8208326697349548, 0.9733326435089111, -0.6961551308631897, 1.2359130382537842, 1.016123652458191, 0.37960928678512573, -0.7910327315330505, 0.8305178880691528, -0.681710422039032, 1.179978847503662, 1.612309455871582, 2.2611520290374756, 0.5030711889266968, -0.19383102655410767, 1.6479824781417847, 1.0612117052078247, -2.990720272064209, -0.9971624612808228, 0.37959304451942444, 0.19079309701919556, -1.100574254989624, 0.481067955493927, 0.052462637424468994, -1.3440005779266357, -2.2458345890045166, -2.2940187454223633, 0.2594972848892212, -0.0036365038249641657, 0.6241155862808228, 0.17991450428962708, 0.4491538107395172, -1.5888632535934448, -0.43791463971138, 0.48026958107948303, -0.5376747846603394, -0.7269388437271118, -0.5639950037002563, 0.9177745580673218, 0.08550316095352173, -1.6909211874008179, -0.1593664288520813, -1.945046067237854, 0.11747381836175919, 0.11504021286964417, 0.3861827552318573, -0.32324039936065674, 1.5218029022216797, 1.1008573770523071, -1.5494080781936646, 0.08502070605754852, -0.3448783755302429, -0.20995081961154938, -0.7122336626052856, 1.6428897380828857, 0.7536509037017822, -0.1201292872428894, 1.0040427446365356, 1.6116914749145508, -0.620776891708374, 2.399034261703491, -1.2577762603759766, -1.4061846733093262, 1.777532935142517, 2.6189382076263428, 2.7859206199645996, 0.5660400390625, 0.3029356300830841, 0.6504842042922974, 0.21820032596588135, -0.018486959859728813, 0.7884832620620728, -0.19771181046962738, -0.936323881149292, -0.01511827576905489, -0.5406556129455566, -1.1015286445617676, 1.3443775177001953, -0.3724566698074341, 0.3907952606678009, 1.5021156072616577, -0.7701283097267151, -0.17736488580703735, -0.12468603253364563, 0.15585896372795105, -1.3426295518875122, 1.3119028806686401, 0.5278161764144897, 0.2867792546749115, -0.12004753202199936, 1.7898250818252563, 1.008543610572815, 0.03728305548429489, 1.2991409301757812, -1.2022160291671753, -0.7328334450721741, 0.002341603161767125, -0.34293362498283386, 0.581356942653656, 0.11351342499256134, -0.62702476978302, 0.5007587671279907, -0.3010985553264618, -0.7595335841178894, 0.5552915334701538, -2.398163080215454, 0.015431134030222893, 1.3643786907196045, 1.1398561000823975, 0.6058658361434937, -0.3717062771320343, -0.1480512022972107, -0.24186226725578308, 0.3196117579936981, 0.20339052379131317, -0.8999033570289612, -0.9676356315612793, -2.0390357971191406, 2.522420644760132, 0.4945257008075714, -0.35300201177597046, -1.386417031288147, 0.4701012074947357, -0.01335793174803257, -0.36741867661476135, -2.530956983566284, 0.25290676951408386, 0.7654199004173279, -0.09954655170440674, -0.937614381313324, -0.238366961479187, -1.1176398992538452, -0.7976953387260437, -0.19087637960910797, -0.31913918256759644, 0.4280889928340912, -1.1421616077423096, 1.622740387916565, -0.9706661701202393, 1.1873899698257446, 1.0793570280075073, -1.476527452468872, -1.0929239988327026, 0.8647229075431824, 0.20412296056747437, -0.1009063571691513, -1.6651405096054077, 1.1372637748718262, -0.2847025990486145, 0.5924732685089111, 0.9879831671714783, 0.4113423228263855, -0.15907227993011475, -0.3668098747730255, -1.2745170593261719, 0.3565274775028229, -0.40114718675613403, -0.9974183440208435, 0.7720060348510742, -0.4195646345615387, 0.0005875991773791611, 2.6125354766845703, 0.744523823261261, -1.725070834159851, 0.26201820373535156, 0.8172483444213867, 0.7498570084571838, -0.2435009479522705, -0.16505968570709229, -0.3477851152420044, -0.7558513879776001, 0.7453415393829346, -0.40279436111450195, 1.6506719589233398, 1.6349109411239624, 1.875791311264038, 0.09109015762805939, -0.4317452907562256, -1.2846310138702393, -0.9132385849952698, -0.7611303329467773, -0.43939101696014404, 0.5585732460021973, -0.36872580647468567, 0.296800434589386, 0.35250964760780334, -1.1763014793395996, 0.5291166305541992, -0.45210909843444824, -0.5705680847167969, -0.4940359890460968, 0.406948447227478, -1.3235509395599365, -1.1910228729248047, -0.3228238523006439, -0.30305975675582886, -1.0624029636383057, 0.0219868291169405, -0.7790688872337341, -1.1233924627304077, -1.3091458082199097, 1.0474183559417725, -0.5856059193611145, -1.4414629936218262, -1.4948309659957886, 1.5005559921264648, -0.026797307655215263, 0.08350856602191925, 0.5340388417243958, -0.7948220372200012, 1.1393810510635376, 0.3416062295436859, -0.49223944544792175, 0.44336721301078796, -0.5734574794769287, 1.373603105545044, -1.069309115409851, 1.1387428045272827, -0.2972065508365631, 0.3407250642776489, 1.1603370904922485, -0.042007800191640854, 1.0114059448242188, 1.5980503559112549, -0.5629977583885193, 1.3305617570877075, 0.2136588841676712, -0.9510893821716309, 0.36324524879455566, 0.15151458978652954, 0.8830363154411316, -0.7676741480827332, 0.5536853671073914, 1.257920265197754, 0.610348105430603, -1.5623140335083008, -2.4155688285827637, 0.7729830741882324, 0.5318771600723267, 1.4271188974380493, -1.1261507272720337, 0.26158463954925537, 0.1252533346414566, 0.2646949589252472, -0.8681456446647644, 0.6732414364814758, 0.2720228433609009, -0.33421704173088074, 0.7420189380645752, -0.28146862983703613, 0.22423356771469116, -2.7443437576293945, -1.4182628393173218, -1.9044348001480103, 0.6063549518585205, -0.02309679053723812, -0.6870383620262146, 0.19043084979057312, 0.2762111723423004, -0.7888706922531128, 0.7369639277458191, -0.27235591411590576, 0.4263085126876831, 0.18926683068275452, -1.816005825996399, -0.49384424090385437, 1.6509244441986084, 0.6083728671073914, 0.38840168714523315, 1.0550340414047241, 1.1015245914459229, -0.7229543328285217, 2.5759527683258057, -0.6409104466438293, 0.40640518069267273, 0.42739659547805786, 0.5811975002288818, -1.5413010120391846, -0.8178297281265259, -0.1797962188720703, -0.24863974750041962, -1.9068968296051025, 2.4959909915924072, 0.7499673962593079, -0.1428779512643814, -1.2034285068511963, -1.0891361236572266, 0.1482142060995102, -2.026411771774292, -0.051724519580602646, 1.1591228246688843, 0.02766694873571396, -0.7148823738098145, 0.7778837084770203, 0.2129213511943817, 0.12774322926998138, 0.9114581346511841, -1.0345860719680786, -1.3565177917480469, 1.6441295146942139, -0.9998168349266052, 1.2935882806777954, 0.23674814403057098, 0.8614662885665894, -0.783667802810669, 0.6719675660133362, -0.8206051588058472, -0.47138306498527527, -1.6883059740066528, 0.08458296209573746, -0.05822557955980301, -0.3666250705718994, -2.230910301208496, 0.10642421245574951, -1.7597174644470215, 1.3662574291229248, 0.2045546919107437, 0.09049583226442337, -0.26018425822257996, 1.4120336771011353, 0.11642459034919739, 0.6506031155586243, -1.2091896533966064, -0.09306346625089645, 1.661049723625183, -0.06261713057756424, -0.5978807210922241, -0.5093896985054016, 0.5916446447372437, -0.3392399251461029, -0.4613134264945984, 0.03779776766896248, 0.278195858001709, -1.249304175376892, 0.21954384446144104, 0.6794196367263794, -1.4246965646743774, 1.6865543127059937, 0.8900444507598877, -0.10951296985149384, -0.20384660363197327, -1.721437692642212, -0.33498308062553406, -2.728656530380249, -1.103948950767517, -2.3561835289001465, 0.7300211191177368, -0.9891335964202881, 0.5836880803108215, -1.3599493503570557, -0.8833511471748352, 0.9037748575210571, 0.7532724142074585, 0.4965084195137024, 0.26424726843833923, 0.3350585699081421, 1.7036598920822144, 0.08093391358852386, -1.0268369913101196, 0.24692147970199585, 0.056739985942840576, -0.39689475297927856, 0.536848247051239, -1.35684335231781, -0.41503962874412537, -0.1977277249097824, 0.9087050557136536, 0.5884194374084473, 0.4985482096672058, -1.9401695728302002, 1.3699676990509033, -2.0383658409118652, 0.9536961913108826, -0.7701449990272522, -1.3914304971694946, 0.8382721543312073, -0.2987412214279175, -1.329707384109497, -0.44801172614097595, -0.35231471061706543, -0.044320546090602875, -0.7768790125846863, 1.0308053493499756, 2.5921080112457275, -0.24769911170005798, 1.525040626525879, -0.8716858625411987, -0.3769412040710449, 0.32005995512008667, 0.4275166690349579, 0.7260560393333435, -0.3073413670063019, 0.41799941658973694, -0.5392520427703857, 0.6447955369949341, 0.06134512275457382, 0.4403577744960785, -0.6779531836509705, -0.24623899161815643, 0.3225143849849701, 0.19009442627429962, 0.6283546686172485, -0.19273363053798676, 0.25754809379577637, -0.309910386800766, 0.9718047380447388, 0.5435396432876587, 0.7607477307319641, -0.8409035205841064, 0.6714769005775452, -0.7232446074485779, -1.2478132247924805, 0.3863251507282257, 0.39000439643859863, -0.07694505155086517, 0.19134798645973206, -0.1341034322977066, -0.47337454557418823, -1.0102097988128662, 0.6297241449356079, -0.640085756778717, -0.10602375119924545, 0.7481393218040466, 1.9716904163360596, 1.8688874244689941, -0.2628154456615448, -1.725611686706543, 0.8985819816589355, 0.7272706627845764, -0.03814278170466423, 0.2082698941230774, 0.3863834738731384, -1.106644868850708, -0.8014233708381653, 2.2431092262268066, 0.16989529132843018, 0.9609218835830688, 0.000799100729636848, -0.6663463711738586, 0.09439800679683685, 0.5764048099517822, -0.3290318250656128, 0.8025665879249573, 0.3910396099090576, -1.5545984506607056, 0.5105664730072021, 0.1753801554441452, -0.701984167098999, -0.3643057644367218, -0.05914969742298126, 1.6878712177276611, 1.4016525745391846, -1.24193274974823, 0.6027080416679382, -1.0898154973983765, -1.0086326599121094, -0.02318495884537697, -0.878517746925354, 1.0604559183120728, 1.2871143817901611, 0.39388588070869446, 0.1511904001235962, 0.8631936311721802, -0.10280901938676834, -0.19680733978748322, 0.8063161969184875, -1.3754937648773193, 0.30561017990112305, 0.16127170622348785, -0.6035142540931702, -0.07657496631145477, -0.329353004693985, 0.0630592405796051, 0.9322423338890076, -0.9381558299064636, -0.6904134154319763, -0.6284483075141907, -0.04489312693476677, 1.3338500261306763, 2.126224994659424, 0.2408825308084488, -0.3934420943260193, 0.49334272742271423, 0.791388213634491, -1.1412405967712402, -0.39325591921806335, 0.039435964077711105, 0.9724579453468323, -0.8936288356781006, 1.4341378211975098, -1.238255262374878, -1.6224180459976196, 0.3917248845100403, 0.5832758545875549, 0.8624775409698486, 0.7153964042663574, -0.40709003806114197, 0.3261271119117737, 0.0165398009121418, -0.9833790063858032, 2.0258209705352783, -0.5236605405807495, 0.7558572888374329, 0.47872066497802734, -0.9877480268478394, 0.36912477016448975, -0.07368732988834381, 0.624628484249115, -0.2476971447467804, 0.09778361022472382, -0.5731683373451233, -0.5300757884979248, -1.572676658630371, -0.2188282310962677, -0.8277327418327332, 1.5404516458511353, 0.8302708268165588, 1.4718552827835083, 1.61202073097229, 0.7739661931991577, -1.4815844297409058, 0.5162050724029541, 0.2644781172275543, 1.8011891841888428, -1.4519561529159546, -1.5263372659683228, 0.23933035135269165, 0.9513812065124512, -1.0456889867782593, -0.8955510854721069, 0.6109092831611633, -1.5943771600723267, 0.278337299823761, 0.47860848903656006, -0.9288837313652039, 1.8960398435592651, 0.11667683720588684, -0.04324406757950783, 0.31686854362487793, -0.4673173725605011, -0.8905638456344604, -1.0728706121444702, -1.1426559686660767, -1.35977303981781, -1.2359620332717896, 0.18824268877506256, -0.5919978022575378, 0.8318367600440979, 1.0087586641311646, 0.30292803049087524, -0.34533458948135376, -0.011904384940862656, -0.2330346405506134, -0.03663936257362366, 0.3689902424812317, 0.4216202199459076, 0.6917945742607117, -0.5478730201721191, 0.13246364891529083, 0.26691684126853943, 0.030335893854498863, -1.6038235425949097, 0.35714462399482727, -0.01517096534371376, -1.665022373199463, -1.0532082319259644, 0.06217042729258537, 1.1235344409942627, 1.5512142181396484, -2.7901670932769775, -1.0322918891906738, -0.6342641115188599, -0.1730288863182068, -0.6591864824295044, 1.4854466915130615, -0.11125081777572632, -0.9974061250686646, 2.0319418907165527, -0.20598021149635315, 1.318631887435913, -0.43081843852996826, -0.40315741300582886, 0.8304653763771057, -2.410717010498047, 0.2466348260641098, -1.8154276609420776, 0.3185839056968689, -0.3132120370864868, -0.5209704041481018, -0.6563488841056824, -0.2989899218082428, -0.654055655002594, 0.07233603298664093, -1.471407413482666, 0.8025375008583069, -1.5507783889770508, -0.03553522378206253, -0.02061687782406807, -0.003030270105227828, 1.2113091945648193, 0.5382137894630432, 1.8387832641601562, -0.7869234085083008, 0.6670613288879395, 0.628365159034729, 1.4321552515029907, -0.9315780401229858, 1.0345666408538818, 0.32253149151802063, -0.10242524743080139, 0.5494701266288757, 0.9189855456352234, -1.4639028310775757, 0.7646172046661377, 0.9436032772064209, 0.9269183278083801, 1.053483486175537, 2.044518232345581, 0.5155923962593079, 2.1896798610687256, -0.20892348885536194, 0.7655683159828186, 0.5565205216407776, -0.1114463359117508, 0.8561815619468689, -1.2487404346466064, -0.8291439414024353, -0.18807031214237213, 0.7356617450714111, 0.5817155241966248, -1.2024074792861938, 1.3887927532196045, -1.1797826290130615, -0.14998039603233337, 0.9880369901657104, 0.6944587230682373, 1.4051562547683716, 0.7775084376335144, 0.4679606556892395, 1.2702665328979492, 0.4047614634037018, 1.3494311571121216, 1.271718144416809, -0.2178107649087906, -0.8244791030883789, -0.41887933015823364, -0.29584893584251404, 0.3275243639945984, 0.040313344448804855, -0.45686137676239014, -2.0700836181640625, -0.03819595277309418, 0.6436101198196411, 0.33721911907196045, -0.9025775194168091, -0.3661715090274811, -0.23836858570575714, -0.911528468132019, -0.2880185842514038, 0.38552126288414, 0.30721426010131836, 0.5180416703224182, 0.8221782445907593, -0.4172995984554291, 0.886712372303009, -0.1674332171678543, -0.0177768524736166, -0.5814812183380127, 1.6545313596725464, 0.679725706577301, -0.31041622161865234, 0.8629662990570068, 1.674985408782959, -1.2277946472167969, 2.1980135440826416, -0.8109731078147888, 0.8231273293495178, -1.9724711179733276, -1.309559941291809, 2.27508807182312, -0.9272946715354919, 0.2419348508119583, 0.31312063336372375, 2.9011383056640625, 0.24627920985221863, 0.8772398233413696, -0.6112500429153442, 1.2045646905899048, 0.5608386993408203, 0.49435117840766907, 2.0961124897003174, -0.6959625482559204, -0.45970574021339417, -0.4339667558670044, -0.08581788092851639, 0.2570995092391968, -0.6114926934242249, 0.04523646458983421, -0.6112130284309387, -0.48755577206611633, -0.7223979830741882, 0.33913278579711914, 0.18548128008842468, -2.4521777629852295, -1.5987414121627808, 0.3033667504787445, -0.3445670008659363, 1.9798297882080078, -0.05613458529114723, -0.13186588883399963, 0.3324011564254761, 0.3650152087211609, 0.3344791531562805, -0.3857026696205139, -0.03590794652700424, -1.1828200817108154, 1.443408727645874, 1.1155714988708496, 0.0010012362617999315, -1.6559298038482666, -0.7216240167617798, 1.5954501628875732, 1.3341246843338013, -1.1010147333145142, -0.34872058033943176, 0.9350273609161377, -2.0950095653533936, 0.01656436361372471, -1.1632977724075317, -1.2416987419128418, 1.012357473373413, -0.5628991723060608, -0.21358707547187805, 1.4560467004776, 2.5174777507781982, -1.4305044412612915, 1.1149053573608398, -1.3182919025421143, -1.3093777894973755, 0.0011006679851561785, 0.6765671968460083, 0.7881054282188416, -0.5089378952980042, 0.296840101480484, -0.8941561579704285, -0.7343091368675232, 0.047891322523355484, 0.3184681236743927, -1.4459025859832764, -1.4989687204360962, -0.7292442917823792, -0.20116347074508667, 1.530311107635498, 1.1393728256225586, -0.3561357259750366, -0.33254796266555786, 0.9629972577095032, 0.14630341529846191, -0.09559207409620285, 1.4884241819381714, -0.5861685872077942, 1.6445553302764893, 1.9611228704452515, 1.4178197383880615, 1.50808584690094, -1.4890435934066772, 0.2314717173576355, -0.872818648815155, -0.07505840063095093, -1.1420767307281494, 0.27104121446609497, 0.9424352645874023, -0.40373748540878296, 0.34123319387435913, -1.7600641250610352, -1.4540075063705444, -0.6384088397026062, -0.7942966222763062, 1.026056170463562, -0.06587810814380646, -0.2806392312049866, -0.7116243839263916, 0.8095828294754028, -0.11671853810548782, -0.5881550312042236, 0.34672367572784424, 0.160521999001503, 0.5892965197563171, -1.4916776418685913, 0.8963766694068909, -0.6827241778373718, -0.024500614032149315, -0.9894051551818848, -1.6872730255126953, -0.6594385504722595, -0.308541476726532, -0.014072065241634846, -0.5040428042411804, 1.1224817037582397, 1.2957850694656372, 1.15244722366333, 1.1348826885223389, 0.5479132533073425, -1.293485164642334, 0.3677203357219696, 1.2752948999404907, -0.2497912347316742, 1.330278992652893, 0.29076269268989563, 0.3311552405357361, 0.8784174919128418, -1.9625208377838135, -1.5460996627807617, 0.2650812864303589, -1.370258092880249, 1.2536271810531616, 0.1295868307352066, -1.792030930519104, 1.4725937843322754, -0.20105348527431488, 0.5116192102432251, 1.521357536315918, -0.2910391688346863, 1.2421095371246338, -0.4537569284439087, 0.6527857780456543, 0.5135305523872375, 1.4597550630569458, 0.46357616782188416, -0.5351050496101379, 1.7268496751785278, 1.0326299667358398, 0.34539592266082764, 1.3051042556762695, 1.871935486793518, -1.0255409479141235, -0.035376593470573425, -0.9846287369728088, -1.0603057146072388, -0.5028448104858398, -0.12570200860500336, 0.6599400043487549, -0.7076341509819031, 1.4027073383331299, 0.4211771488189697, 1.3285908699035645, -0.7801350355148315, -0.1910468339920044, -0.12370067089796066, -0.42893344163894653, 1.2325800657272339, -0.4706036150455475, 2.2837026119232178, -0.24366945028305054, 0.14897210896015167, -0.6595665216445923, -0.6917656660079956, 2.045525312423706, -1.292431116104126, 0.015405761078000069, 0.4762183129787445, -0.9604791402816772, -0.6617187261581421, -0.2683577239513397, -0.809273362159729, -0.6942176222801208, -0.17813259363174438, 1.6429964303970337, -0.39596542716026306, 1.3390454053878784, -0.766420304775238, 0.38024717569351196, -0.9703456163406372, 0.18533112108707428, 1.047251582145691, 0.7454778552055359, 0.23853321373462677, -1.8228676319122314, -0.4308728873729706, 0.4530297815799713, -1.4868539571762085, -0.6985461711883545, -1.1651504039764404, -0.686052143573761, -0.4714808464050293, 0.6205915212631226, 1.4990322589874268, -0.3686031103134155, -0.79231196641922, 1.2563449144363403, 0.10851392894983292, -2.459810733795166, 0.30170243978500366, -0.20671091973781586, 1.6758321523666382, -1.4613375663757324, -0.14119669795036316, 0.10392801463603973, -0.7014848589897156, -0.15321005880832672, -0.12433602660894394, -0.15141433477401733, 0.8318303823471069, -0.8021817803382874, 1.8225877285003662, 0.7779521942138672, -0.4248092770576477, -2.2865853309631348, -0.46109580993652344, -1.8232094049453735, -0.076579250395298, 0.16977380216121674, 0.7634274363517761, -0.7395334839820862, 0.14647150039672852, -0.7848125696182251, -0.1764587163925171, -0.20493212342262268, -0.26571235060691833, 0.8522582054138184, 0.15534934401512146, -0.036962252110242844, 0.4560301899909973, -0.04315761849284172, 1.2970815896987915, -1.22165846824646, -0.016576141119003296, 0.751987874507904, 0.4766705334186554, 0.354955792427063, 0.8557693362236023, 0.2891994118690491, 1.3144018650054932, 1.1102557182312012, 0.8262470960617065, 1.2218581438064575, -1.962410569190979, 0.5651550889015198, 1.2845571041107178, 0.30156585574150085, -0.4600291848182678, -2.10709810256958, 0.1423986703157425, -0.11111659556627274, 0.8509344458580017, 0.9556859731674194, -0.368997722864151, 2.247647762298584, 0.7186610698699951, -0.004051464144140482, -0.9789024591445923, 0.19406349956989288, 0.22601348161697388, -0.40563663840293884, -0.8251099586486816, 1.3309770822525024, -2.0753135681152344, 0.5750907063484192, 2.0249218940734863, -0.5673462152481079, -0.030344894155859947, -1.4953227043151855, 0.1777791678905487, -0.30183014273643494, -0.503796398639679, -0.4739357829093933, -0.15363676846027374, -0.21338628232479095, -1.8113117218017578, -1.2037031650543213, -1.3251007795333862, 0.7760940194129944, 1.0644023418426514, 0.6427947282791138, -1.0932992696762085, -0.6984819173812866, 1.122403860092163, -2.2600252628326416, 0.9452264308929443, -1.2600643634796143, 1.2121670246124268, 0.5993772745132446, 1.3756672143936157, 0.19370198249816895, 1.9802978038787842, 0.8167816400527954, -1.0230679512023926, -1.6973514556884766, -0.2314705103635788, 0.26212358474731445, 0.8787882924079895, 1.0165153741836548, 0.40176016092300415, 1.7317194938659668, -0.08290103822946548, 1.0242929458618164, 1.3253763914108276, 0.6513789892196655, -0.6055877804756165, 0.5365862250328064, 0.5562601089477539, -0.5959451198577881, 0.351687490940094, 0.18753071129322052, -0.5303658246994019, 0.7125289440155029, -0.5242571830749512, -0.3381110727787018, -0.5035694241523743, -1.149113416671753, -0.15697430074214935, 1.081063985824585, -2.0024254322052, -1.8418142795562744, -1.9781153202056885, -0.8402003645896912, -0.8775861263275146, -0.6190028190612793, -2.26745343208313, 0.9368484020233154, -0.2949865758419037, -0.3405754268169403, -0.5021991729736328, 1.381574273109436, 0.3600991666316986, 1.8652044534683228, 0.09465361386537552, -0.6352403163909912, -0.8008337020874023, -0.8591916561126709, 0.45757386088371277, -0.9190744161605835, -0.4417457580566406, -0.11954781413078308, -0.7608819603919983, 0.15085169672966003, 1.4015005826950073, -1.3243433237075806, -0.9423112869262695, 1.226912021636963, 0.3111180365085602, -0.9026349782943726, 0.949293851852417, -0.5407161116600037, 1.0376267433166504, -0.7141301035881042, -1.2319570779800415, -1.2601393461227417, -1.7318270206451416, 1.0585459470748901, -1.601859211921692, -0.5336698293685913, -0.13257011771202087, -0.056687429547309875, 0.38338086009025574, -1.3293006420135498, -1.612933874130249, -0.2500111162662506, 1.0263755321502686, 0.39479726552963257, -0.7648924589157104, -0.152725487947464, 0.21034309267997742, -0.8770590424537659, 1.0525811910629272, 1.7439098358154297, 1.5718921422958374, 0.21220844984054565, -0.10548363626003265, -0.5613682270050049, -1.136991024017334, -0.756804347038269, 2.056398868560791, 0.06870166957378387, -1.537335753440857, 1.6514654159545898, 0.5839237570762634, 0.013269409537315369, -0.9342123866081238, 1.931440830230713], [-0.6048924326896667, 1.0672816038131714, 0.572936475276947, -0.32370173931121826, 0.17945514619350433, -0.8525956273078918, -1.1529223918914795, 1.185410499572754, 1.0264495611190796, 1.600136399269104, -0.87882000207901, 0.7721822261810303, -0.44330307841300964, 0.36623284220695496, -1.1509538888931274, 0.5794327259063721, -0.016241254284977913, -1.1180922985076904, -1.1820603609085083, 0.003669855184853077, -1.513693928718567, -1.1392524242401123, 1.3841384649276733, -1.1746671199798584, 0.1631310135126114, 1.231211543083191, 0.025145787745714188, -1.5377023220062256, -0.3834497928619385, 0.9196379780769348, 0.036861203610897064, 1.3915002346038818, 0.3851134479045868, -1.668062686920166, -0.822398841381073, -2.061192750930786, -0.2876094579696655, -0.5488269329071045, 0.4396502673625946, -0.4236496090888977, 0.14760692417621613, -1.2549505233764648, -3.154545307159424, 0.22472929954528809, -0.9363893866539001, 1.4837368726730347, 0.5288062691688538, -0.6563381552696228, -0.39679500460624695, 0.45572981238365173, 0.08568337559700012, -0.9836881756782532, -1.2574608325958252, -0.09728152304887772, -0.5089544057846069, -0.5994446873664856, -1.5026068687438965, 1.2066880464553833, -0.09684354066848755, -0.35884928703308105, 1.1201331615447998, -0.6021353006362915, 1.2878797054290771, 1.2518947124481201, -0.8423033952713013, -0.10857886075973511, 0.12676940858364105, 0.5019431114196777, -1.443459153175354, 0.8514502644538879, 0.02745934948325157, 0.44703733921051025, -0.455047070980072, -0.6094759106636047, -0.4522240161895752, -0.5381829738616943, -0.5552094578742981, -0.276933878660202, -1.4274433851242065, -0.5163991451263428, 0.23205862939357758, -0.20867031812667847, -0.9044613838195801, 0.5012626051902771, -0.5735819935798645, 0.2906450033187866, -0.9297453165054321, -0.6326114535331726, -0.5995848178863525, 0.09993188083171844, -0.15366707742214203, 1.4646093845367432, -1.1273809671401978, 0.3890581429004669, -0.33290067315101624, 0.6449093818664551, 0.12459001690149307, -1.192186713218689, -0.6738468408584595, 0.2924705445766449, -2.271787405014038, 0.7131136655807495, 0.063264861702919, -1.5622955560684204, -0.014052852988243103, -0.6954230666160583, -1.6760199069976807, -1.2683736085891724, -0.9481760859489441, -1.4923069477081299, -0.34253910183906555, -0.9784225821495056, -0.2702288329601288, -0.47310885787010193, 0.7216318249702454, 0.890203058719635, 0.00469933170825243, -0.4070826768875122, -1.4216654300689697, -0.9884669780731201, -2.558117389678955, 0.6660535335540771, 0.18538983166217804, -0.6855436563491821, -0.566491425037384, -0.003183020744472742, -0.24285882711410522, 1.0309720039367676, 0.25196927785873413, -0.2533664107322693, 0.7112076282501221, 0.6922743916511536, 0.5100396871566772, 1.0838795900344849, 0.8138446807861328, -1.2359721660614014, -0.08958947658538818, -0.6842652559280396, -0.09075421839952469, 0.5494885444641113, -0.5050209760665894, 0.08489324897527695, -2.474017858505249, 1.5725369453430176, -0.4918677508831024, -0.7020537853240967, 1.0868486166000366, 1.0221271514892578, -0.6736887693405151, 0.6002839207649231, -1.0837955474853516, -1.5321580171585083, 0.5556957721710205, 0.007378286682069302, -0.852266252040863, 1.6220693588256836, 0.5345299243927002, 0.7606961727142334, -1.4633371829986572, 0.1724061518907547, -2.667069435119629, 1.6228822469711304, 1.2413989305496216, 0.11753599345684052, 0.5760089755058289, 1.6452031135559082, 0.07473684847354889, -1.4287426471710205, 0.696950376033783, -1.2374484539031982, 0.5806010365486145, 0.3201310634613037, 0.33723223209381104, -0.5046305656433105, -0.684029757976532, 0.19624152779579163, -0.9223666191101074, -0.41278088092803955, -1.970475673675537, -0.852979302406311, 0.22905582189559937, -0.5242326855659485, 2.2041642665863037, -1.7611470222473145, 0.9339399337768555, 2.8848981857299805, 0.09254154562950134, -0.1743105798959732, 0.10694847255945206, -0.6779168248176575, -0.7987124919891357, 1.7863712310791016, -1.6252506971359253, -0.2612919211387634, -0.6667687296867371, 1.733974575996399, 0.006645489484071732, 0.8910254836082458, 0.7888156771659851, -0.12481264024972916, -0.46870332956314087, -0.21232588589191437, -0.9795792102813721, -0.39234328269958496, -0.0012696210760623217, -1.342939019203186, 0.8798927068710327, 0.12050000578165054, -1.01173996925354, -0.22508929669857025, 0.5595653057098389, 0.43344181776046753, 0.9702905416488647, 1.2858306169509888, 0.12786905467510223, 0.3349243700504303, -0.22071675956249237, 0.916418731212616, -1.1375786066055298, 0.4873107671737671, -0.17969268560409546, -1.0232195854187012, 0.3548549711704254, -0.39877817034721375, 0.32423144578933716, 0.2174626588821411, -1.2022687196731567, 0.10214655846357346, -0.01016845740377903, 1.7810999155044556, 0.2815586030483246, -0.6854052543640137, 0.3286224901676178, -0.6066768765449524, -0.07067696005105972, 1.710169792175293, 3.421898603439331, -0.038697149604558945, -1.4283051490783691, -0.4043133854866028, 0.9840437769889832, -1.085388422012329, 0.8894424438476562, -0.8008836507797241, -1.1527892351150513, 0.046718012541532516, 0.5102701783180237, 1.4153518676757812, -1.195508599281311, -0.3867207169532776, -0.7538740038871765, 0.6575149297714233, 0.9412298202514648, -0.16443681716918945, -0.9971596598625183, 0.651339590549469, 0.2264811098575592, 0.05413511395454407, -0.7713850140571594, -0.06353061646223068, 1.30628502368927, -0.4417154788970947, -0.08770358562469482, -0.5740633010864258, 0.11404923349618912, -0.03552252799272537, -1.3282946348190308, -2.7778067588806152, 1.2200994491577148, -0.9957273602485657, -1.0510461330413818, 1.2925459146499634, -0.0852045789361, 1.0170488357543945, 0.16120463609695435, -0.739158570766449, -0.16196952760219574, 0.8210294246673584, -1.289869785308838, 0.933826208114624, 1.4791066646575928, 1.004533290863037, 0.9018874168395996, 1.2673265933990479, 0.3740825057029724, 1.5629745721817017, -0.45185452699661255, -0.7882235050201416, 0.1084575429558754, 1.3212891817092896, 0.33362963795661926, 0.17757423222064972, 0.507019579410553, 0.6335111856460571, 0.7349975109100342, -0.6711990833282471, -0.5548383593559265, -1.379023551940918, -0.6601822972297668, 0.16255909204483032, -0.3316298723220825, -0.26158443093299866, 1.5607728958129883, 0.08065047860145569, -0.6294614672660828, 1.0657910108566284, 1.0118522644042969, -1.199210524559021, 0.5790577530860901, 0.21653589606285095, 0.26368847489356995, 0.3002973198890686, -0.9982654452323914, -0.24257120490074158, -1.3534564971923828, 0.12734048068523407, -0.516869068145752, -0.3010302484035492, 0.347796767950058, -1.1043756008148193, 2.4703309535980225, 1.0791683197021484, 0.6703661680221558, -1.512483835220337, 0.49435415863990784, 2.4610767364501953, -1.951162576675415, 1.2367702722549438, 0.3433800935745239, 0.5170912742614746, 1.895810842514038, -1.165426254272461, 0.7699620127677917, -1.5266226530075073, -0.13176016509532928, -0.3308474123477936, 1.430197834968567, -0.7472608089447021, 1.0688457489013672, -1.1251686811447144, -1.4217661619186401, -0.8020355701446533, 0.4105157256126404, 1.9974255561828613, -0.30890461802482605, -1.1748589277267456, -0.6002870202064514, -0.3074638247489929, 0.5410036444664001, -1.315524697303772, -0.2961340844631195, -0.3699248135089874, -1.2593129873275757, 0.9720093607902527, 0.4684862494468689, 0.5794658064842224, -0.2601275146007538, -0.07242736965417862, 0.3747689723968506, 0.05308753252029419, 0.9515610933303833, 0.4489434063434601, 1.4881417751312256, 0.13883085548877716, -0.7766234278678894, -3.8588099479675293, -1.5754774808883667, -1.4863582849502563, 0.3236722946166992, -0.3800241947174072, 0.3961840867996216, 0.28137868642807007, 0.9766940474510193, 1.0061910152435303, 0.06454172730445862, -0.22650912404060364, 0.055502999573946, 0.9225278496742249, -0.11746063828468323, 0.8555907011032104, 1.3868356943130493, -0.4350171387195587, 1.426255226135254, 1.1551223993301392, -0.5739699602127075, -0.22122204303741455, -0.9835771918296814, -1.7135977745056152, 0.9047988057136536, 0.18308423459529877, 0.44050687551498413, 0.5098392367362976, 0.4966093599796295, -1.0097461938858032, 0.23123599588871002, -0.011083096265792847, 0.0982021763920784, 0.637128472328186, 0.29519960284233093, -0.4139224886894226, 0.03000619076192379, 1.1418758630752563, -0.0855608806014061, 0.5908373594284058, -0.20491890609264374, 0.47586166858673096, -1.24283766746521, 1.1768052577972412, -0.16006259620189667, -0.15913960337638855, -0.5810856223106384, 0.1516725718975067, -0.6877871751785278, -1.4104740619659424, 1.0076862573623657, 0.8474195003509521, -0.357601135969162, -1.554969072341919, 2.0985658168792725, -0.21213987469673157, -0.49932435154914856, -0.37269794940948486, -0.00958630908280611, 2.223322868347168, 0.4156275987625122, -0.24547158181667328, 0.20463436841964722, 0.17428292334079742, -1.1823481321334839, 0.3991439938545227, 0.9169221520423889, 2.0015833377838135, -0.8871133327484131, 0.5665391087532043, 0.7934778928756714, -0.10107244551181793, -1.0313814878463745, 1.2647660970687866, 0.1018199771642685, 0.8058528900146484, -0.7074965238571167, 1.3068033456802368, 0.943051815032959, -0.9453481435775757, -0.6928479671478271, -0.636740505695343, 0.6846993565559387, -0.466348797082901, -1.3912408351898193, -1.1466219425201416, -0.3481879234313965, -0.015354576520621777, 0.05906146392226219, 1.5448176860809326, 1.7279187440872192, -0.3065202534198761, 0.20531414449214935, 0.06998692452907562, 0.4606334865093231, 0.7288039326667786, -0.06482888013124466, 2.0473134517669678, 0.2011926919221878, -1.7367135286331177, -0.4427637457847595, 0.29880473017692566, 0.5251479744911194, -1.6224493980407715, 0.7197126746177673, 0.4972587823867798, 0.1447225958108902, 0.6510595083236694, 1.250623345375061, -0.2602372467517853, 1.2098897695541382, -0.4981442093849182, 0.46084433794021606, -0.582507848739624, 0.5756481289863586, 0.2907596826553345, 0.008581660687923431, -0.4276493787765503, 1.908896565437317, 0.0192447230219841, -0.33982419967651367, -1.5145000219345093, 0.5950973033905029, -0.02887204848229885, 1.8635529279708862, 1.0263736248016357, -1.377046823501587, 1.1940410137176514, -1.1143102645874023, 0.86463463306427, 0.01233473140746355, 0.0450616180896759, -0.6265271306037903, 0.45613715052604675, -1.1609389781951904, -0.5045589804649353, 0.7470059394836426, -0.09067574143409729, -1.1796954870224, -0.07088606804609299, -0.9307645559310913, 1.0751731395721436, -0.6959414482116699, -0.36979034543037415, 1.5663944482803345, 0.04617277905344963, 0.3167904317378998, 0.27769267559051514, 1.3870153427124023, 0.09093616902828217, 0.2768498361110687, 0.696449339389801, 0.7878154516220093, 0.6589887142181396, -1.629304051399231, -0.10591432452201843, 0.9815344214439392, 0.7138463258743286, -0.3144158124923706, 1.0734422206878662, -0.8760342001914978, 0.8796478509902954, 0.8040220737457275, -0.305189847946167, 0.4623279571533203, -1.7052758932113647, -1.1005915403366089, -0.10096850246191025, -1.0552492141723633, 0.5266407132148743, 1.9710278511047363, -0.1608082354068756, 1.2164816856384277, -1.197380542755127, 0.4908224940299988, 1.7795047760009766, 2.399003028869629, -0.6059079170227051, -1.5756906270980835, -0.36784061789512634, -0.17432667315006256, 0.18102537095546722, 0.8292747139930725, 1.0385475158691406, 0.8230208158493042, -0.4887421727180481, 2.1854162216186523, -0.4964261054992676, -0.5906607508659363, -1.382049798965454, -0.17304567992687225, -0.18331696093082428, 1.1369595527648926, -0.7806711196899414, 0.6210167407989502, 0.0637180507183075, 0.22691312432289124, 0.11732962727546692, 0.21504147350788116, -0.45631203055381775, -3.1793384552001953, 0.49118858575820923, 0.03540673851966858, 0.07681204378604889, -0.6461513638496399, 0.183327317237854, -0.7556594014167786, 1.372027039527893, -0.3216913342475891, -1.4856609106063843, -0.5014200806617737, -0.7436128854751587, 1.4283933639526367, -1.343651294708252, 0.29165586829185486, 0.22105905413627625, 0.7901477217674255, 1.5221631526947021, 0.2406957745552063, 1.585528016090393, 0.7774235010147095, -0.8537325263023376, -0.9604440331459045, 1.3371506929397583, -0.16971512138843536, -2.2432711124420166, 1.1032648086547852, 2.5382931232452393, 0.3023228645324707, 0.5687649250030518, -0.8683596849441528, 0.14622484147548676, 0.7976483106613159, 0.05056535825133324, -0.7961946129798889, -0.3536182940006256, -1.4070461988449097, 1.7708908319473267, -1.0852129459381104, -3.5183393955230713, 0.02016928419470787, 0.42546889185905457, -1.1114692687988281, 1.0496668815612793, -0.795022189617157, 1.3797216415405273, 0.05232308804988861, 1.6395716667175293, 0.0621647983789444, -0.9368909597396851, -1.8542648553848267, -0.7287235260009766, 0.07213025540113449, -0.43645673990249634, -1.6455546617507935, 0.006603694520890713, 0.6273900270462036, -1.236451506614685, -0.3454868495464325, 0.47401297092437744, -0.6190978288650513, -0.5513700246810913, 0.12740595638751984, -1.6536228656768799, -1.0080071687698364, -1.2829551696777344, -0.03954131528735161, 0.1921212375164032, 0.7611227035522461, 0.5211397409439087, -0.08427055925130844, -0.4849209487438202, -1.2064433097839355, -2.04980731010437, 0.4799499809741974, 1.599987506866455, -0.014215096831321716, -0.15191206336021423, -0.9356381297111511, -2.3668901920318604, 0.3484707772731781, -0.32909438014030457, -0.05016271770000458, 0.9242838025093079, -0.5784378051757812, -0.6290610432624817, 0.8285934329032898, 0.14643897116184235, 0.548831045627594, -0.7445397973060608, -0.40653616189956665, -1.2896487712860107, -0.229952871799469, -0.5193965435028076, -1.1305526494979858, -1.9643579721450806, -0.40327197313308716, 0.2534632086753845, 0.13248372077941895, -0.43831929564476013, -0.08307559043169022, 0.6381871104240417, 0.3135899007320404, 0.4643273651599884, 0.09263107180595398, 1.6288659572601318, -0.18295180797576904, -0.06876076757907867, -0.5800513029098511, -0.020404592156410217, -2.4068875312805176, 0.7669846415519714, -1.1812009811401367, 2.9176692962646484, -0.38861531019210815, 0.8065870404243469, -2.0035996437072754, -0.5488218665122986, 0.04311109706759453, -0.7504963278770447, -0.09942571818828583, 1.2978277206420898, -1.7964410781860352, -0.596439778804779, 0.02951355278491974, 0.41722410917282104, 1.133638620376587, 0.7035780549049377, 1.4671269655227661, -1.1317203044891357, -0.9474859833717346, 0.6718462109565735, 0.9180710911750793, -0.6604084968566895, -1.035545825958252, 0.6517460942268372, 0.11318086087703705, -0.05193738266825676, -1.1225306987762451, 0.002816244261339307, -0.5948958992958069, 1.324905514717102, 0.404579758644104, 0.05393962189555168, -0.013878281228244305, -0.620235025882721, -0.5242798328399658, 1.1285004615783691, 0.3565636873245239, 0.597438633441925, 0.5656847357749939, 0.02298092283308506, -1.8854373693466187, -0.049423422664403915, -0.93991619348526, 0.6226325631141663, 0.1064097061753273, 0.36355090141296387, -0.12794117629528046, 0.8277251720428467, 0.4159744381904602, 0.6714949011802673, -0.9674736261367798, 0.08565650880336761, -0.5685977935791016, 0.005065281875431538, -0.9474833011627197, -0.9212361574172974, -2.0635881423950195, 0.9582410454750061, 0.16044606268405914, 0.300382137298584, -0.6322963833808899, 0.7413957715034485, -0.9868881702423096, -0.49602341651916504, 1.0079952478408813, -1.2781903743743896, 1.289918065071106, -2.0938432216644287, 0.18657377362251282, 1.4370311498641968, 0.9311280250549316, 1.146734356880188, 0.7308520078659058, 0.4838472902774811, -0.27687880396842957, 0.01455780677497387, 0.33425599336624146, -1.2182419300079346, -0.6863893866539001, -0.8012773394584656, -0.44671621918678284, -0.7146114110946655, -0.6955307722091675, -1.0058218240737915, -1.3043110370635986, 0.245184987783432, -1.2740356922149658, 1.5111932754516602, -0.5620495080947876, -0.019616946578025818, -0.379804790019989, 0.01841713674366474, -2.155801773071289, -1.0018291473388672, 0.8000099658966064, -0.24601857364177704, -2.3636064529418945, -0.34800973534584045, -0.8332822918891907, 0.43889307975769043, 0.7105778455734253, -1.1814653873443604, 0.10593484342098236, -1.7400776147842407, 1.2370457649230957, -0.3287266492843628, 1.1312233209609985, 1.3735946416854858, -0.4806321859359741, -1.4933711290359497, 0.5107904672622681, 2.401230573654175, 1.317006230354309, -0.6677036881446838, -0.238358736038208, -0.3984977900981903, 0.4215932786464691, 0.814850926399231, 0.186804860830307, 0.21779339015483856, 1.3311349153518677, -0.6202183365821838, -0.6098599433898926, -0.1264338195323944, 0.20481595396995544, -1.2107439041137695, 0.1482531577348709, -0.19436313211917877, -0.42767685651779175, -1.6076812744140625, -0.9636031985282898, -0.6905481815338135, 0.9618186354637146, -2.1030826568603516, 0.5180495977401733, -0.21056108176708221, -1.1547660827636719, 0.31337377429008484, -0.1806221753358841, 1.443036437034607, 0.5470974445343018, 0.9682824015617371, -0.44453519582748413, -0.5519509315490723, 0.19966824352741241, -1.3097397089004517, -0.6161136627197266, -0.12357664108276367, 1.0435113906860352, -0.8545147776603699, 0.32343292236328125, -0.29447901248931885, -1.066628336906433, 1.419542908668518, 0.0923217311501503, 0.4636234641075134, -0.7473699450492859, -1.9306756258010864, -0.26646891236305237, 0.5634230971336365, 0.33925679326057434, 1.3640637397766113, -0.9882133603096008, 0.5042385458946228, 0.5655681490898132, -1.5980420112609863, 0.22036273777484894, -0.09527259320020676, 0.2805165946483612, -0.7757614254951477, 0.10465554147958755, 1.113071084022522, 0.03489145264029503, -0.19560548663139343, 0.41281774640083313, 0.3977622091770172, -2.74236798286438, 1.5851536989212036, -1.5417675971984863, -1.0946115255355835, 0.9417212605476379, 0.254431813955307, -1.5194804668426514, -0.12549453973770142, -0.9061012268066406, -0.12956446409225464, -0.8103036880493164, 0.4758748710155487, -0.04282696172595024, -0.14584805071353912, -1.700512170791626, -0.36319199204444885, 0.4988962709903717, 0.8440316319465637, -0.7312138676643372, 1.1603566408157349, 0.5272396206855774, 1.7723301649093628, -0.6494309902191162, -0.5011158585548401, -0.5245953798294067, -0.18062762916088104, 0.31383517384529114, 1.0689817667007446, -0.4003216028213501, -0.39194217324256897, -0.46753305196762085, 1.1859816312789917, -0.9527038335800171, 0.5975185036659241, 0.949597179889679, 0.17123445868492126, -0.8153741955757141, -0.8645976185798645, -0.3532622456550598, 0.8327357769012451, -0.6690240502357483, 1.2637553215026855, -1.3236424922943115, 0.4944436848163605, 0.2748706340789795, 0.6166650056838989, 0.8831577301025391, -1.0371840000152588, 0.36528342962265015, -0.7402912974357605, 0.13112543523311615, 0.32555559277534485, 1.2869127988815308, -0.3766785264015198, -0.6566351652145386, 0.32249218225479126, 1.5466532707214355, -1.0635583400726318, -0.4207737147808075, 0.4611225724220276, 0.47372788190841675, 0.27158066630363464, -0.3089463412761688, -0.08229338377714157, 0.8432304859161377, 0.20830583572387695, -0.5107311606407166, -0.29545947909355164, 1.3639353513717651, 1.0655814409255981, -0.6310731172561646, 0.960345983505249, -0.7920587062835693, 0.405510276556015, -1.1858868598937988, -0.38299962878227234, 0.5429894328117371, -2.1163270473480225, 1.9369131326675415, -0.4310317635536194, 1.3654882907867432, 0.3339083194732666, -0.4287220537662506, 1.507341742515564, 1.7960015535354614, -0.015447293408215046, 1.1248339414596558, 0.9802564978599548, 0.4840482473373413, 0.6212546825408936, -0.18693099915981293, -0.3102124333381653, 0.05985529348254204, -1.7464065551757812, -0.6678661108016968, 1.0465242862701416, 2.110689878463745, 0.4481508433818817, -0.2585754692554474, 0.2851215600967407, 0.5596829056739807, 0.21682803332805634, 0.023696962743997574, 0.5878390073776245, 0.48875370621681213, -0.1260247379541397, -0.48762214183807373, 0.5033674240112305, -1.0853379964828491, 0.7847335934638977, 1.792698621749878, -0.7616906762123108, 0.13598211109638214, 1.4461991786956787, 1.0764200687408447, 0.14788822829723358, -0.9087243676185608, -1.0936940908432007, 0.04441294074058533, -0.18596285581588745, -1.1931864023208618, -1.282382845878601, 0.259628027677536, -1.5171700716018677, 1.4587758779525757, -1.9096150398254395, -0.9262901544570923, -2.24910044670105, -0.12972737848758698, 1.9188041687011719, 2.380185842514038, -1.692959189414978, 0.8182203769683838, -1.3151612281799316, -0.41345131397247314, 1.565937876701355, -0.2342417985200882, -0.9556906223297119, -0.34231865406036377, -0.2008361965417862, 0.36706823110580444, 1.0274279117584229, -2.003976345062256, 0.7371289730072021, 0.6097795963287354, -0.6061792373657227, 1.9763009548187256, -0.14047309756278992, 0.2303047925233841, 1.7913720607757568, 1.3089497089385986, -3.0955960750579834, -0.17247001826763153, 0.02491859346628189, -0.017459671944379807, -0.42473292350769043, -0.02997819520533085, 0.632080614566803, -1.4598006010055542, 0.5485647320747375, -1.6558507680892944, -1.8700305223464966, -0.7743011116981506, 1.4661473035812378, -0.20760884881019592, 0.7004449367523193, 0.9902521967887878, 0.3136014938354492, 1.658128023147583, -1.3075268268585205, 1.4801132678985596, 0.234266459941864, -0.8796433210372925, -0.8953028917312622, 0.4672904908657074, -0.5418696999549866, -1.732369065284729, 1.3546810150146484, 0.015468903817236423, -2.9957165718078613, -0.4104456603527069, -0.4902763068675995, 0.15373004972934723, 0.43182551860809326, 1.04190993309021, 1.7091755867004395, 0.0017992270877584815, 1.3470909595489502, -0.22254221141338348, -0.406759649515152, -1.773023247718811, 0.10748101025819778, 1.3963344097137451, -1.86652410030365, -1.142651915550232, 1.356999397277832, 0.7613124251365662, -0.6209527254104614, -1.440730094909668, 0.4140510559082031, 0.4887051284313202, -0.6968927979469299, -0.4553440809249878, 0.06801202893257141, -0.019026877358555794, -0.5462578535079956, -0.8443208932876587, 1.1179152727127075, 1.5432944297790527, -0.25812578201293945, 1.599092960357666, -0.9057267308235168, 0.5702471733093262, 0.27128639817237854, 1.6756631135940552, 1.0360767841339111, 0.3826914131641388, 1.1709381341934204, 0.1972947120666504, 1.9031174182891846, 0.28676632046699524, -0.7356426119804382, -0.10325874388217926, 2.3040339946746826, -1.6073226928710938, 0.24216100573539734, -0.10404189676046371, 1.4355988502502441, 1.0273211002349854, -1.2177658081054688, 0.0791739821434021, 1.5286978483200073, -1.4552159309387207, 0.040291618555784225, 0.6737257242202759, 0.27041739225387573, 1.3807024955749512, -1.1347761154174805, 0.6516919136047363, -0.40050211548805237, -0.5006235837936401, 1.080000877380371, 0.5548986196517944, -0.6820834875106812, 1.6911580562591553, 0.17666438221931458, -0.058466553688049316, 0.5947124361991882, 0.9239019155502319, -2.5950675010681152, 0.37720632553100586, -0.9520605206489563, 0.6253083348274231, 0.5153207182884216, 0.4277484714984894, 0.2127860188484192, -1.8855681419372559, -0.856162428855896, 0.99708491563797, -1.8668148517608643, 0.7224071621894836, 0.062165766954422, -0.7448499798774719, -0.12218717485666275, 1.0642788410186768, 0.1397504210472107, -0.5816985368728638, 0.027277598157525063, -0.5463348031044006, 1.8544023036956787, -0.06944993138313293, -2.013730764389038, 0.13333585858345032, -0.7891876697540283, -0.7319703698158264, -0.3315882086753845, 0.9141337871551514, -2.0008294582366943, 0.8186711072921753, -0.7763440012931824, -1.1210321187973022, 0.26468685269355774, 0.27670323848724365, -0.39399585127830505, -0.10894502699375153, 1.3489034175872803, -0.004893677309155464, -0.34050729870796204, -0.3462473154067993, -1.1671302318572998, 0.7082438468933105, 0.8183193206787109, 0.3174891769886017, 0.2938477098941803, -0.781488835811615, 0.26725631952285767, 0.1888238787651062, 0.04963786527514458, -1.1176930665969849, 0.22060997784137726, -0.5300784111022949, 1.4477564096450806, 0.6998538970947266, 0.9310654997825623, -1.4290825128555298, -0.2662014365196228, 0.19091863930225372, 0.1150236651301384, -0.04908188432455063, -1.5236979722976685, 0.48171788454055786, -0.8339086771011353, 0.01677565835416317, -1.3943355083465576, 1.3113077878952026, 1.6692273616790771, -1.3131736516952515, -0.681498646736145, -0.4124644100666046, 0.08559661358594894, -0.8956969976425171, 1.2744390964508057, 1.3371896743774414, 0.4857947528362274, -2.167224884033203, 0.3463950753211975, 1.6107046604156494, -0.11327936500310898, 2.274832248687744, -0.5884646773338318, -1.9549521207809448, -1.4984073638916016, 0.6297513246536255, -1.105666160583496, -1.3807646036148071, -0.1379888653755188, -0.1593020111322403, -0.241481751203537, 0.19952073693275452, -0.5583531856536865, 0.21355518698692322, 1.1472214460372925, 0.09589362144470215, 0.013761705718934536, -0.06310173869132996, 1.3542077541351318, -0.12368177622556686, 0.35953059792518616, 1.3277533054351807, -1.2793346643447876, -0.901216983795166, -0.5898561477661133, -1.6354602575302124, -0.42064884305000305, 2.0850658416748047, -0.6404658555984497, 0.1024647131562233, -0.9841644167900085, 1.0747826099395752, -1.1117228269577026, -0.894547164440155, -0.8522820472717285, 0.7534477114677429, 0.760883092880249, 0.22489185631275177, -0.1711755096912384, -0.868792712688446, -0.33782657980918884, 0.4724214971065521, 2.182100772857666, 0.6868495941162109, -1.0482409000396729, 1.0447970628738403, -1.1641466617584229, -1.86550772190094, -0.5617289543151855, -0.07117243111133575, -0.07533412426710129, -0.21284662187099457, 0.4072490930557251, -0.07028056681156158, -1.6625839471817017, -0.16659438610076904, -0.7080901861190796, -0.6484414339065552, -1.169939398765564, 0.22519896924495697, -0.08138730376958847, 0.650836706161499, 1.1150116920471191, -0.25267985463142395, 0.1652422696352005, -1.3441189527511597, 0.11400029808282852, -1.6966509819030762, -0.5077621340751648, -0.48062723875045776, -0.8731759786605835, 1.119374394416809, -1.9010401964187622, -0.6408876776695251, 1.2815147638320923, 0.15611419081687927, -0.7400689721107483, 0.27433833479881287, 0.936660885810852, 0.7242171168327332, 0.6175431609153748, 1.9086834192276, -0.25974851846694946, 0.1385425627231598, -0.023990949615836143, -0.23756927251815796, -2.1259360313415527, -0.32049399614334106, -0.15024331212043762, -0.3608245253562927, 0.5768154263496399, -0.5540599822998047, 0.26776373386383057, -1.719703197479248, 0.14394906163215637, -1.081215500831604, 0.17036256194114685, 0.40589243173599243, -2.3527636528015137, -1.6114215850830078, 0.7050009965896606, -0.44117265939712524, -0.42966601252555847, -1.0271985530853271, 0.37171855568885803, -1.1522469520568848, 0.44208696484565735, 0.4335145652294159, -1.7407727241516113, -0.3370290696620941, 0.21148373186588287, -0.4829317033290863, 0.07729410380125046, 0.30870866775512695, -0.681856095790863, -0.3260222375392914, -0.1917933076620102, -0.05996216833591461, -2.0737240314483643, -0.18627892434597015, 0.6370722055435181, 0.6122722029685974, -0.09317100793123245, -0.7480283379554749, -0.2386651635169983, -1.1321666240692139, 0.5700559616088867, -1.4001120328903198, 0.1406087577342987, 0.474210649728775, 0.15674325823783875, -0.1458495855331421, 0.13307718932628632, 0.6963822245597839, -1.7186793088912964, 0.009875228628516197, 0.3227415978908539, 0.9632384181022644, 2.7643444538116455, -0.5162498950958252, -2.1065802574157715, 0.49069446325302124, 0.03150046616792679, 1.8279035091400146, 0.7034949064254761, 1.3583139181137085, 0.37576380372047424, 0.7934231758117676, 0.6216420531272888, 0.19205817580223083, -0.9962937235832214, 1.4941778182983398, 0.12085223197937012, -0.6388447880744934, -0.24097485840320587, -2.573073387145996, -0.1482698619365692, 0.171449214220047, 3.316683053970337, 0.3077757656574249, -0.15222759544849396, 1.3770220279693604, -1.1816864013671875, 0.7106139659881592, -2.0089244842529297, -2.4147088527679443, -0.39944955706596375, -0.1155841201543808, 1.4202091693878174, 0.5233402252197266, -0.8295188546180725, -0.006014975719153881, -1.7815983295440674, 0.005187766160815954, -0.8870216608047485, 0.2928994596004486, 0.789825975894928, 1.3758805990219116, 0.22131381928920746, -2.144683599472046, 0.285995751619339, -0.031467389315366745, 0.6455472707748413, 1.6100486516952515, -0.2466275691986084, -0.6018617153167725, -1.4819873571395874, 1.4424307346343994, -2.729872465133667, -0.3653844892978668, -0.6662546992301941, 0.17840978503227234, -0.34516486525535583, -0.38202139735221863, 0.24221442639827728, 0.7860012650489807, -0.6506248712539673, -0.8426485061645508, 0.30770546197891235, -0.022564848884940147, 0.5227665901184082, 1.572705626487732, 2.4943361282348633, -0.8178525567054749, 0.2865931987762451, 0.5007011890411377, -1.1514784097671509, -0.5238680243492126, -1.1304388046264648, 1.3163232803344727, 0.6085134148597717, 0.8067268133163452, 0.14117634296417236, 0.9845248460769653, 0.23846526443958282, -0.41726720333099365, 0.4573950171470642, 0.5898370742797852, 0.33160194754600525, -1.0724539756774902, 2.9752209186553955, 0.7208528518676758, 1.2557622194290161, -0.49311596155166626, 1.4105477333068848, 0.46011894941329956, -1.2572896480560303, -0.14772258698940277, 0.016276797279715538, -0.7979232668876648, 1.0022320747375488, -1.1240856647491455, 0.7048333883285522, -0.10076915472745895, 0.24663236737251282, -1.3343405723571777, 1.105987548828125, 0.4139169156551361, -1.731584072113037, -0.7175979614257812, 0.7035714983940125, -0.11790040135383606, 0.3306277096271515, 1.0909146070480347, -0.46722742915153503, 1.0120465755462646, 1.2273461818695068, 0.6629428863525391, -1.510943055152893, 0.1846684217453003, -2.144765853881836, -0.21792960166931152, 0.1319376677274704, 1.073747992515564, 0.06347164511680603, -0.6050848960876465, -0.1708536148071289, 0.084959976375103, -0.27807819843292236, 0.39854156970977783, -0.5661934614181519, 0.27315735816955566, -0.26485854387283325, 1.4431346654891968, 0.17782242596149445, -0.2976946234703064, -1.9005324840545654, 0.11482953280210495, 1.2239748239517212, -1.5316314697265625, 1.0310425758361816, 0.1876678466796875, 0.39419904351234436, -1.1563990116119385, -0.1035427376627922, -1.2471747398376465, -0.4118236005306244, -0.3439452052116394, 2.224923610687256, -0.7942802309989929, -1.4162802696228027, -1.1200414896011353, 0.4962819814682007, 1.1348737478256226, 0.7745913863182068, -0.3158101439476013, 0.09986361116170883, -0.3970739245414734, -0.7913556694984436, -0.6810292601585388, 1.6323460340499878, -0.4271000921726227, -0.5388809442520142, 2.351201295852661, -1.306422233581543, -1.044838786125183, -1.379202127456665, 0.6891613006591797, 0.296015202999115, 2.065931797027588, 0.3062851130962372, 1.9877763986587524, -1.1512700319290161, 1.177899718284607, -1.1852452754974365, -0.027835670858621597, -0.8602609634399414, 0.7915619611740112, 0.8231462240219116, 0.5794814229011536, 0.8510296940803528, -1.4171758890151978, 1.9848687648773193, -1.1990538835525513, 0.07346196472644806, -0.3352828919887543, 1.6001946926116943, 0.06134066730737686, -1.806888461112976, -0.15892599523067474, 0.8433523774147034, -0.6469285488128662, -0.4304233193397522, -0.28293588757514954, -0.22612115740776062, 0.35199445486068726, 1.2941102981567383, 0.27390024065971375, -0.07514093816280365, -1.2538599967956543, -0.5175175070762634, 1.3026087284088135, 0.6386626958847046, -0.8179963827133179, 0.8934165239334106, 0.270875483751297, 1.4138355255126953, 0.13102608919143677, 1.0285249948501587, -1.0682311058044434, 0.587615966796875, 0.5866691470146179, 0.38806474208831787, 1.2208139896392822, -1.3104383945465088, 1.0014629364013672, -0.3746562600135803, 0.46560415625572205, 2.3848354816436768, -0.21037019789218903, -0.7779048681259155, 1.1367032527923584, -1.8802027702331543, -0.7271706461906433, 0.6281037330627441, 3.8951799869537354, -1.4415254592895508, 0.07384820282459259, -1.1029409170150757, 1.1167281866073608, 0.3482324481010437, -0.5324061512947083, -0.9689401388168335, 0.36030688881874084, 0.48722097277641296, -0.4005841314792633, 1.0496028661727905, -0.5684462785720825, 0.3774309456348419, 1.4562283754348755, 1.104371428489685, -0.3123181164264679, 1.8294581174850464, 1.5757300853729248, -0.006454753689467907, -1.980378270149231, -1.238029956817627, 0.18736310303211212, -0.45487308502197266, 0.32351431250572205, -0.08245985209941864, -1.486598253250122, -0.8384917974472046, -0.3592882454395294, -0.5287365317344666, -0.979055643081665, -1.5522745847702026, 0.4092247188091278, -1.2923427820205688, 2.0102763175964355, 2.215688705444336, 1.0677378177642822, -0.9568677544593811, -0.47596925497055054, -0.8027870655059814, 0.04778105020523071, 1.1503698825836182, -0.6608319878578186, 1.4929077625274658, -0.49801164865493774, 0.17494216561317444, -1.1759644746780396, 0.3645150363445282, -0.27776405215263367, -1.4029340744018555, 0.39865127205848694, 0.675130307674408, -0.656346321105957, -0.702448844909668, 1.6415200233459473, -0.5084965825080872, -0.36069250106811523, 0.11898955702781677, 0.7862244844436646, -0.09691452980041504, -1.143175482749939, -0.5000118613243103, 1.1293659210205078, 0.6560012102127075, -0.31070902943611145, 1.782885193824768, -0.0069650644436478615, 0.1424720287322998, -1.0486483573913574, -1.2124388217926025, 0.9600092768669128, 0.537534773349762, -1.251759648323059, 1.0131534337997437, 1.447327733039856, 0.059790704399347305, 1.4883867502212524, 0.08993520587682724, 0.3949829339981079, -0.4169076681137085, -0.2890087962150574, 1.1928513050079346, 1.1451830863952637, 0.3714292645454407, -1.1425403356552124, 0.44296780228614807, -0.7138357162475586, 1.5955795049667358, -0.35486647486686707, 0.5339936017990112, 0.12477133423089981, 1.1975783109664917, -2.3724515438079834, 0.24435582756996155, -1.0958466529846191, 0.2928323447704315, 1.5648455619812012, -1.2009854316711426, 0.8813348412513733, -0.2773814797401428, -0.5005130767822266, 0.6867440342903137, 0.2118251919746399, -0.12322524935007095, 1.9826217889785767, -1.145755410194397, -0.22859446704387665, 3.604684591293335, -1.4373843669891357, 0.772379457950592, -0.6092328429222107, 0.06750696897506714, -0.3095141351222992, 0.6245197653770447, -0.35581663250923157, -0.13727357983589172, 1.4217344522476196, -1.2104283571243286, 2.0454509258270264, -0.5803948640823364, -0.10636813938617706, -0.4218827188014984, 1.4824409484863281, 0.7054974436759949, -2.1924054622650146, 0.5441398024559021, -1.9162882566452026, -0.5544621348381042, 1.3309712409973145, 0.18007005751132965, 2.2517242431640625, 2.4910473823547363, -1.0805060863494873, 0.3209165930747986, -0.9335441589355469, -0.41741645336151123, -0.11232152581214905, 0.48865529894828796, 1.175599455833435, 0.4392700791358948, 0.8445943593978882, -0.7398507595062256, -0.5559155344963074, -0.636444091796875, 1.047346591949463, 0.9933192729949951, 0.8822092413902283, 0.04949119687080383, -0.14599482715129852, 0.2525019645690918, -0.4183259904384613, -0.6824774146080017, 1.497314214706421, -0.2608589828014374, 0.8835432529449463, 0.19389598071575165, 1.059818983078003, -0.502737283706665, -1.2910526990890503, 1.1958664655685425, -0.5915634036064148, -1.1181436777114868, -0.47000810503959656, -0.8052650094032288, 0.4148249328136444, 0.8660780787467957, -1.4012877941131592, -0.2923221290111542, -1.3894400596618652, -1.9591577053070068, 0.7050453424453735, 2.6571414470672607, 1.312703013420105, 0.38385605812072754, -0.3367820382118225, 0.4530242681503296, 0.4373312294483185, 0.8152128458023071, -0.03468915447592735, -0.013732122257351875, -0.49105268716812134, 1.4724433422088623, 0.9089435935020447, 0.2995913624763489, 1.1791952848434448, 1.4920992851257324, 0.6770675182342529, 0.49997881054878235, -0.8497505784034729, 0.9803609251976013, 0.034276191145181656, -0.4010004699230194, -1.8287802934646606, 0.7590003609657288, -1.379363775253296, -1.6245776414871216, 0.28633224964141846, -1.4974477291107178, -1.2417933940887451, 0.21217358112335205, -0.6894461512565613, -0.9608235359191895, -1.1850329637527466, 0.7999968528747559, -0.6201037764549255, -1.122988224029541, 0.7125548720359802, 0.22236420214176178, -0.2895807921886444, 1.0026336908340454, 0.784105658531189, 0.03008013591170311, 0.3072757422924042, 1.3392733335494995, -0.12258594483137131, -0.8936765789985657, 0.7433303594589233, -0.16845369338989258, 0.12131516635417938, 0.18496324121952057, -0.654911458492279, 0.30188456177711487, -0.42678582668304443, -0.23575575649738312, 0.3589494228363037, 0.6025773882865906, -0.6493143439292908, -0.8563149571418762, 1.1142319440841675, 0.8198487758636475, -0.04324236512184143, -1.652282476425171, 0.6411001682281494, -0.40609362721443176, -0.9044705629348755, 0.22482433915138245, -2.7308950424194336, 1.1046257019042969, 0.907794713973999, 0.0806288942694664, 0.4117581248283386, -0.8733644485473633, -1.2364230155944824, 0.08156616240739822, 0.7645106315612793, -0.6528745293617249, -0.030100757256150246, 1.265447974205017, 2.1426072120666504, 1.3581230640411377, 0.116150863468647, 0.7877913117408752, -0.3286707401275635, 0.3458934426307678, 1.3467735052108765, 1.3385647535324097, 0.20998935401439667, 0.32170894742012024, 0.16956932842731476, -0.7925630211830139, 0.5242617726325989, -0.10631414502859116, -1.0827090740203857, -1.6375068426132202, 0.48288148641586304, -0.803531289100647, -1.0132180452346802, 1.29811429977417, 0.1873767226934433, 0.8815244436264038, -0.7161128520965576, 0.5481923818588257, 1.0197089910507202, -0.4714199900627136, 0.02526792697608471, -1.2250548601150513, -1.5654163360595703, -0.27876999974250793, -0.46487894654273987, 1.1986351013183594, 0.3584314286708832, 1.1891340017318726, 0.21943840384483337, 0.5150176882743835, 0.6298869848251343, 0.39668190479278564, -0.05228143557906151, 0.622465193271637, -0.2835554778575897, -0.9415332674980164, 0.5086902976036072, 0.2724078893661499, -0.10606119781732559, -1.3000670671463013, 1.4606213569641113, -0.46219030022621155, 0.31142106652259827, 0.07282377779483795, -0.3367276191711426, -0.688790500164032, 0.7996199131011963, -1.3554577827453613, -0.5048348903656006, -2.0208475589752197, 0.11850598454475403, -1.0674632787704468, -1.5883629322052002, 0.3980311453342438, -1.5714049339294434, 0.43278613686561584, -2.0809104442596436, -0.2051125019788742, -0.35989585518836975, 0.11712305247783661, 0.5715286135673523, 0.8464290499687195, -1.5462977886199951, -1.0239875316619873, -1.572026252746582, -0.603074312210083, -0.07411863654851913, 2.05603289604187, 0.3721557557582855, 0.045183464884757996, -1.8954051733016968, -0.8078248500823975, 0.9532628059387207, 1.0559494495391846, 1.776705265045166, 0.29437974095344543, -0.5756502747535706, 0.6568178534507751, -0.18495531380176544, 1.135961890220642, -1.0881716012954712, -0.0034091221168637276, -0.563947319984436, -0.5291699767112732, 0.1071852445602417, 0.3250007629394531, -0.43686044216156006, 1.276414394378662, -0.9642280340194702, -0.7099754810333252, -0.02225763536989689, 0.42330172657966614, 0.1548680067062378, 1.323359489440918, -0.694091260433197, -0.9000592231750488, 0.8955675959587097, 2.0031681060791016, 2.624734878540039, 0.13355673849582672, -0.3104034960269928, -0.37740370631217957, -0.13482384383678436, -0.7902166843414307, -0.6831839084625244, -0.48111552000045776, -0.262035071849823, -0.2640865743160248, -2.005192279815674, -1.875773549079895, -0.2763879895210266, -0.7944663166999817, -1.080204725265503, 0.6924601197242737, -0.29174110293388367, 0.3912378251552582, -0.041552573442459106, -0.16005036234855652, -0.6084777116775513, 2.5965707302093506, -1.5130506753921509, -0.6056954860687256, -2.369499444961548, 1.3539643287658691, -1.2118521928787231, -1.3011659383773804, 0.3826597034931183, -1.5340611934661865, 0.2389734983444214, -0.33254653215408325, -1.5010005235671997, -0.7575637102127075, 1.1031455993652344, 0.9101022481918335, -0.9505981802940369, -1.4868266582489014, 1.9633252620697021, 0.7217409014701843, 0.30672770738601685, -0.997714638710022, -2.0734450817108154, -0.11202076077461243, 0.12488551437854767, 1.0245002508163452, 0.42863699793815613, 0.528116762638092, 1.3080034255981445, -1.5278171300888062, 1.919623851776123, 0.4959258437156677, 0.12095663696527481, 0.5196322202682495, -0.29900509119033813, 0.3361918330192566, 0.2839690148830414, 0.4465295374393463, -0.05092586949467659, -0.8555454611778259, -1.6460453271865845, 0.10152405500411987, -1.3317557573318481, -0.314518004655838, -0.7752056121826172, -1.1083126068115234, -0.5000112652778625, 0.23155519366264343, 0.5611233115196228, 0.7338753342628479, -1.622550129890442, -0.9389917254447937, 0.035244908183813095, -1.1797308921813965, -0.6662994623184204, 1.1676814556121826, -0.6258893013000488, 0.5544326305389404, -0.681096076965332, -1.2623435258865356, -0.08421242982149124, -2.2536988258361816, -0.4822683334350586, 1.6927404403686523, -0.21966537833213806, -0.2001703381538391, -0.0865413174033165, -0.9166780710220337, 0.9427521228790283, 0.19039952754974365, -0.07481645792722702, -1.0573432445526123, -0.30098026990890503, 1.716792345046997, -0.34848254919052124, -0.9061607122421265, 2.5236215591430664, 0.10167840868234634, 0.5047874450683594, 0.6649054288864136, -0.5757799744606018, 1.3828668594360352, -0.5211526155471802, -0.3251323699951172, 0.011323836632072926, 1.3809870481491089, 0.11140138655900955, 0.7814438939094543, 0.5855236649513245, 0.9533921480178833, -0.16726993024349213, 1.153329849243164, 0.3512688875198364, 0.38519003987312317, -0.5427424311637878, 0.9902942180633545, 1.0104303359985352, 0.2798618674278259, 1.173209309577942, -0.2772959768772125, -1.1648772954940796, 1.2085901498794556, 0.1145763099193573, -1.3109455108642578, -0.3180660605430603, -0.9047395586967468, 0.27065765857696533, -0.9364945888519287, -0.4393521845340729, 1.6514842510223389, 1.667094111442566, -0.18198037147521973, -0.4321858286857605, 0.45815807580947876, 0.3413971960544586, 0.0652264952659607, 2.123831033706665, -0.42766350507736206, -0.03235451504588127, -0.4948698580265045, 0.05431307479739189, 0.6059316396713257, 0.13769429922103882, -0.020339053124189377, -1.4308364391326904, 0.019055569544434547, -1.2187808752059937, 0.9680576920509338, -1.1031975746154785, -2.0783638954162598, -0.02011813037097454, -0.06501846760511398, -0.44833120703697205, 1.363263487815857, 1.084141731262207, -1.2722997665405273, 0.9743479490280151, -1.3183284997940063, 0.19605061411857605, 1.7268632650375366, 0.7069140672683716, -1.721009612083435], [0.6619996428489685, -0.8866222500801086, 0.41101452708244324, 2.4708311557769775, -1.824022650718689, 0.5399981141090393, 0.034846432507038116, 0.8844312429428101, 0.08421251177787781, 1.4402753114700317, 0.2782691419124603, 0.7881254553794861, 0.5833323001861572, 1.781589388847351, -0.8588569164276123, -2.760145425796509, 0.07667725533246994, -0.42387011647224426, -0.22013132274150848, 0.7713388800621033, -1.5100812911987305, -0.6168362498283386, -1.7994531393051147, 0.600342333316803, -1.3575700521469116, 0.4619484543800354, -0.5799825191497803, 1.5182934999465942, -1.0223990678787231, -0.02787042036652565, 0.6499579548835754, -2.904609441757202, -0.22161255776882172, -0.5670746564865112, -1.5384238958358765, 0.9714698791503906, -0.8273763656616211, -2.0468034744262695, 0.3183854818344116, -2.017253875732422, 0.36364424228668213, 0.286395788192749, 0.24953900277614594, -0.7110103368759155, -0.7484239935874939, -0.01447204127907753, -0.4855893850326538, -1.1035010814666748, -0.4948117733001709, -1.824047565460205, 0.24049384891986847, -0.1822335571050644, -0.21221823990345, -0.4869106411933899, -1.0319548845291138, 0.7223691940307617, 0.044541023671627045, -2.052497386932373, -0.7463255524635315, 0.8055336475372314, -1.5063420534133911, -0.5763577222824097, -2.293750047683716, -2.058345079421997, -1.3288533687591553, -0.21652008593082428, -0.3122829794883728, 1.44906747341156, -2.1376984119415283, -1.275055170059204, -1.0112589597702026, -0.41463208198547363, 0.6393238306045532, -3.2177717685699463, 0.35578253865242004, 0.014632991515100002, -0.9535866379737854, 0.48617836833000183, -1.7530217170715332, 1.3732340335845947, 0.792574405670166, -0.4905249774456024, -1.6394742727279663, 1.9283677339553833, 0.8786792755126953, -0.08882483094930649, 0.8997902274131775, 0.7959469556808472, -0.8712732791900635, -0.677191436290741, 0.17616891860961914, -0.2803868055343628, -0.906497061252594, 1.6346049308776855, -0.32381439208984375, -0.8852679133415222, -0.012504799291491508, -2.128446102142334, -0.03240909054875374, 0.8793876767158508, 1.444596529006958, 0.9719283580780029, -0.0030460369307547808, 0.01897713541984558, 2.0809247493743896, -0.25083571672439575, 0.37849652767181396, -0.39351126551628113, 0.18563896417617798, 0.4885169267654419, -0.4903903603553772, -1.041397213935852, -1.4324499368667603, -0.14141719043254852, 2.43464732170105, -0.10005983710289001, -0.2901793420314789, 1.3386757373809814, 0.23202303051948547, 0.6114540100097656, -0.8191595077514648, -1.349794626235962, 0.5160609483718872, 1.7848066091537476, -1.167994737625122, 0.24331307411193848, -0.032333701848983765, 0.8177049160003662, -0.9536857008934021, 0.8679696917533875, 0.7401037216186523, 0.8046101331710815, -0.7195499539375305, -0.4864857792854309, 0.3163372278213501, 0.6903538107872009, -0.8196724653244019, -1.3692020177841187, 1.1746472120285034, 0.6364595890045166, -1.0075299739837646, -0.1741437017917633, 1.6562329530715942, -0.18639792501926422, 0.9432669878005981, -1.3665826320648193, -1.009348750114441, 0.2214932143688202, -0.4768133759498596, 0.9138425588607788, 0.476118266582489, 0.18518729507923126, 0.10468529909849167, 1.257497787475586, 0.8449550867080688, -0.5641387701034546, -0.7537343502044678, -1.1668646335601807, 1.3029896020889282, -0.38531753420829773, -0.7532225251197815, -0.634294867515564, -0.40361374616622925, -2.2512712478637695, 0.9126419425010681, -0.11551022529602051, -0.06366574764251709, 0.9866911172866821, 0.5338941812515259, -0.2549329996109009, -1.1390407085418701, -0.8153218626976013, -0.8539777398109436, 1.3885825872421265, -0.3255142569541931, -0.1970892697572708, -0.6866592764854431, -1.3798233270645142, 1.0988811254501343, -0.6404657363891602, -0.7673888206481934, -1.0472785234451294, 0.4147431254386902, -0.6887547373771667, 0.22191756963729858, 0.2077377736568451, -0.8219533562660217, -0.9625399112701416, 0.24268676340579987, 2.837109327316284, -1.068132758140564, 1.3790605068206787, 0.43720677495002747, -1.0069326162338257, -0.6449155211448669, 0.24612492322921753, 0.1476808488368988, 0.008185057900846004, 0.8613427877426147, -1.5349613428115845, -0.7751877903938293, 1.6844546794891357, -0.8030772805213928, 0.29843947291374207, -1.930291771888733, -0.9010791182518005, -0.8170926570892334, -0.6768062114715576, 0.01201371755450964, 0.6766217350959778, 1.412551760673523, 0.28773897886276245, -0.3471069931983948, 0.19069650769233704, 0.11885573714971542, 0.7250738739967346, 0.4702909290790558, 1.7434954643249512, -0.28564590215682983, 0.03282555937767029, 0.08014843612909317, 0.08307401090860367, 0.2535732686519623, -0.9877802133560181, 0.022043736651539803, 0.9175390601158142, 0.46438443660736084, -0.45586034655570984, 1.538116455078125, -1.283488154411316, -0.9305224418640137, 1.9929829835891724, 1.126391887664795, -1.4117034673690796, 1.1881781816482544, 0.6714290976524353, -1.808830738067627, 1.142409324645996, 0.2822708487510681, 0.18218623101711273, 0.9448617100715637, 2.473639965057373, -0.8688157796859741, -1.262658953666687, -1.3878376483917236, -0.47217491269111633, 1.1230374574661255, -0.6212962865829468, 0.3307074308395386, 1.0439839363098145, -1.4057849645614624, 0.43464112281799316, -0.002508084522560239, -1.276875615119934, -0.8684223890304565, -0.16528162360191345, -0.9246544241905212, 0.08679904043674469, -0.014975812286138535, -0.6939355731010437, -0.45794209837913513, -0.46018603444099426, -1.0704562664031982, 1.4477187395095825, 1.1294291019439697, 0.8255512714385986, 0.22872932255268097, -1.2399582862854004, 0.9401032328605652, -2.4139692783355713, -0.1889781504869461, -0.6260027289390564, 1.358211874961853, 0.21887999773025513, 0.5473523139953613, -0.041061826050281525, 1.2489486932754517, 0.9331409931182861, 1.0132728815078735, -0.10775909572839737, 0.6958739757537842, 1.145447015762329, -0.11952465772628784, 0.4685141444206238, -0.8225229382514954, -0.8857889771461487, -0.910696268081665, -0.4372776448726654, -0.05853540822863579, 0.02299490012228489, 1.4275234937667847, -0.17280720174312592, 1.7994319200515747, -0.9275234937667847, 1.1812000274658203, 0.19145506620407104, -0.7629795670509338, -0.2525668740272522, -1.3192716836929321, 0.10228264331817627, 0.44302839040756226, -0.16642078757286072, 0.05300586670637131, 0.11396213620901108, -0.71207594871521, -0.9374402165412903, 0.05628815293312073, 1.1227517127990723, -0.6418845057487488, 0.5061951875686646, -0.456849604845047, -0.2367333322763443, -2.565356731414795, 0.32707667350769043, 0.4092998802661896, 1.1501356363296509, -0.9901719093322754, -0.9068946838378906, -0.11141753941774368, 0.5536948442459106, -0.43336939811706543, -0.8731375932693481, 0.5486322641372681, 0.42538291215896606, 0.6818071007728577, 0.4721629023551941, 1.4286972284317017, 0.3379497230052948, 0.9850512742996216, 0.0814095288515091, 0.749422013759613, 0.10772936791181564, 0.629765510559082, -0.5242597460746765, 0.43478572368621826, 0.9189218878746033, -0.5852558612823486, -1.1605435609817505, 1.1176751852035522, -0.2949734032154083, 0.9427164196968079, -0.38914087414741516, 0.4101933240890503, 0.545665442943573, -0.0342414565384388, -1.429816722869873, -0.6670463681221008, 0.0514661967754364, -0.7221978306770325, -1.2574628591537476, 0.5221929550170898, -0.16562867164611816, 0.7229332327842712, -2.236616849899292, 0.23415453732013702, -1.275391697883606, 0.4184778928756714, -0.14969392120838165, -0.5198180675506592, -1.4727232456207275, 1.6788406372070312, -0.6588079929351807, -0.08770766109228134, 0.32621267437934875, -1.6475353240966797, -0.27866169810295105, 0.8501201272010803, 0.953073263168335, 0.7022020220756531, 0.011526559479534626, 2.1629467010498047, 0.9592130184173584, 0.8975202441215515, -0.5279248952865601, -0.08084236085414886, 0.33462876081466675, 2.1792752742767334, -0.367777556180954, 0.861236572265625, -1.157289981842041, -1.4114103317260742, 0.5872267484664917, 1.0026047229766846, -0.6812427639961243, -0.28513607382774353, 1.5121862888336182, 0.6926243901252747, -0.506996214389801, -2.5418686866760254, 1.3552281856536865, -0.8866596817970276, 0.4005894064903259, 0.3771810829639435, -0.4129440486431122, 1.0383638143539429, 0.19841602444648743, -0.6177276968955994, -0.8565919995307922, 0.8567361235618591, -0.5260417461395264, 0.35938847064971924, -1.5300816297531128, 0.5970977544784546, -0.21402327716350555, -0.27574628591537476, 0.05809110030531883, -0.9776644110679626, -0.6123318672180176, 0.37303346395492554, -0.6257758140563965, 0.5859447121620178, 0.8182570338249207, -0.6785538196563721, 0.12907929718494415, 0.6320725679397583, -0.23551566898822784, -0.725436270236969, 0.38612475991249084, 0.34452423453330994, 0.6899393796920776, -0.9864625930786133, -0.36709854006767273, 0.0821027010679245, 0.051455143839120865, 1.5596507787704468, -0.8833462595939636, -0.15428729355335236, 0.15618778765201569, -0.0983809158205986, 0.5611186027526855, 0.41731610894203186, 0.2913181781768799, 0.11763554811477661, -0.04900100827217102, 1.2189888954162598, 0.5416969060897827, 0.5172250866889954, -0.05230427533388138, 0.5046544671058655, -1.4335386753082275, -0.16149553656578064, 0.22109492123126984, 0.6530585289001465, 0.41882285475730896, -0.588454008102417, 1.222365140914917, -0.1634228229522705, 0.37081411480903625, 0.5043091177940369, 0.11055352538824081, -0.46149006485939026, -1.8895725011825562, 1.195906162261963, 3.389528274536133, 0.9999552965164185, -1.0903759002685547, 0.42025789618492126, 0.03024187870323658, -0.6334434151649475, 1.4518569707870483, 1.14973783493042, -0.41641345620155334, 0.5676371455192566, 1.0025426149368286, -1.0161997079849243, 1.1422929763793945, 0.6295526623725891, -0.08876153081655502, -0.9345558881759644, -0.12854422628879547, -1.5781633853912354, 1.6927515268325806, -0.9805238842964172, -0.7815959453582764, -0.3875480592250824, 0.27701887488365173, 2.718024492263794, -1.4894001483917236, -1.4872167110443115, 0.17167815566062927, -1.9310628175735474, 0.7447414398193359, -0.09206115454435349, 0.5012480020523071, 1.6153583526611328, -0.7106732726097107, -1.0714247226715088, -0.3828875422477722, -0.33942344784736633, 0.43704932928085327, -1.936264157295227, -0.29596054553985596, 0.9990659952163696, 0.09754446148872375, 0.03724028170108795, -0.18800859153270721, -0.3031668961048126, -0.1122124046087265, -0.5438412427902222, -1.2636257410049438, 0.47832992672920227, 0.9505593776702881, 0.021567286923527718, -0.07776110619306564, 1.7172131538391113, 0.5023089051246643, 0.1979142278432846, -1.398112416267395, 0.2682730555534363, 1.7831480503082275, 0.11366946250200272, 0.272259920835495, -1.0325052738189697, 1.5871989727020264, 0.6210691928863525, 1.3612704277038574, -1.2233332395553589, 1.9516206979751587, 0.6789239645004272, -1.5701498985290527, 1.6164206266403198, -0.08294989168643951, 0.2855644226074219, 1.6049385070800781, -0.9718878865242004, 0.29757067561149597, 1.2416455745697021, 0.458136647939682, 2.821261405944824, -2.512336492538452, -1.4872057437896729, 0.061482809484004974, 1.0094486474990845, 1.0512769222259521, 1.3664218187332153, -0.6384488344192505, -0.3794335126876831, 0.555394172668457, -0.6518183350563049, -0.1468769609928131, 0.6992217302322388, 0.20190145075321198, -0.2424805462360382, 1.5168596506118774, 1.0963716506958008, 1.456230640411377, 1.2663342952728271, -0.4292473793029785, -0.8196485638618469, -0.33759135007858276, 0.3509633243083954, -1.6284375190734863, -2.547055959701538, -0.4231833517551422, -0.1772289276123047, -1.3569666147232056, -0.18597589433193207, -0.9928005337715149, 1.3305915594100952, 0.9271768927574158, -1.5995101928710938, 2.2089169025421143, -0.9473727345466614, 0.5198854804039001, -2.046422004699707, 0.5007243752479553, 0.12236213684082031, -1.4477736949920654, -0.5945714712142944, 1.0278514623641968, -0.16864466667175293, 0.5707201957702637, 0.65818852186203, 1.9605475664138794, -2.0870745182037354, -0.8524959683418274, 0.38465791940689087, 0.32742840051651, -0.1811063438653946, -0.47365814447402954, -0.6333526968955994, 0.15890097618103027, 0.6114633679389954, 0.3574913740158081, -0.16000138223171234, 0.08756344020366669, 1.0906052589416504, 0.7983962297439575, -1.0461217164993286, 0.5024144649505615, 0.14385752379894257, 0.16486164927482605, 0.3369944989681244, -0.1936742663383484, -0.9592806696891785, -1.0897698402404785, 0.22415418922901154, 1.1040314435958862, -1.5659571886062622, 0.7936680912971497, 1.5459990501403809, -0.800246000289917, 0.8762633204460144, -1.0429620742797852, 1.5666674375534058, -0.4938337504863739, -0.5414044857025146, -0.6385431289672852, 0.018364664167165756, 2.653437376022339, -2.2175827026367188, 0.2895137369632721, -0.1048998162150383, -0.9978194236755371, -0.868779718875885, 1.155331015586853, 0.4863128364086151, -0.4770711064338684, -0.567508339881897, 0.0968523845076561, -1.3269332647323608, -0.13786368072032928, -1.5432990789413452, -1.5224215984344482, -0.4471109211444855, -0.21711575984954834, 0.20872606337070465, 0.9200054407119751, -1.8469492197036743, -0.5590258836746216, 1.2824726104736328, 1.259722113609314, 1.0172802209854126, -0.7773696184158325, -0.09955763816833496, 2.3910727500915527, -1.4676679372787476, 0.9852247834205627, -0.9917003512382507, -0.6798258423805237, -0.6228575110435486, 0.1900539994239807, -0.40341585874557495, 0.567341685295105, -0.02474307455122471, -0.6705964803695679, -1.5345309972763062, 0.7466726303100586, 0.001456658123061061, -1.2414551973342896, -2.060249090194702, 0.965747058391571, -0.22335708141326904, 0.21742866933345795, 0.2224883884191513, 0.43173667788505554, 1.4694314002990723, -1.1987240314483643, -0.45193275809288025, -0.512615978717804, -0.7703729271888733, -0.607766330242157, -0.12517164647579193, 0.8203443884849548, 0.29979532957077026, -0.8364574313163757, -0.13001306354999542, -0.22350548207759857, -0.7087458968162537, 0.188140407204628, -0.07012560963630676, 1.371842384338379, -1.6931804418563843, 0.1784827560186386, -1.2857359647750854, -0.40249016880989075, -0.17144426703453064, 0.17360436916351318, 0.18454353511333466, 0.5535039901733398, -0.9022446870803833, -1.5507969856262207, 0.03607610613107681, -0.06431157141923904, 1.9971519708633423, 1.8211405277252197, -0.2778424918651581, 0.4855740964412689, -1.2482630014419556, -1.7193548679351807, -1.460984230041504, -0.8902710676193237, -0.39450597763061523, 0.1731700748205185, -2.0743229389190674, 1.3903324604034424, 0.19542337954044342, -0.5151554346084595, 1.0556708574295044, 1.256229043006897, -0.13287726044654846, 0.2408735603094101, 0.19925914704799652, 1.6821551322937012, -1.3559837341308594, 0.6352510452270508, -0.8116410374641418, 0.7017511129379272, 0.6531105637550354, 1.2999213933944702, 1.1288714408874512, -0.278816282749176, 0.34917157888412476, 0.5952394604682922, 0.40133213996887207, 1.289003849029541, 0.10551979392766953, -0.6771418452262878, -2.2332091331481934, -1.1148349046707153, -1.8527799844741821, 0.6191512942314148, -1.0997902154922485, 1.3233098983764648, 0.7929868698120117, -0.557638943195343, 0.021238092333078384, -1.076134443283081, -0.33967524766921997, 0.2892995774745941, -0.11902952194213867, -1.298685073852539, 1.5520133972167969, 0.8863130807876587, 0.3059353530406952, -0.2978214621543884, -0.22916367650032043, -0.05346318706870079, -1.1023502349853516, 0.5182521343231201, 2.486701726913452, -0.6268317699432373, 1.2018542289733887, 0.39740830659866333, -0.179508239030838, 0.6852656006813049, -0.45860642194747925, -1.7965569496154785, -0.13219602406024933, -0.38964349031448364, 0.01946292817592621, 1.4200291633605957, -2.5400030612945557, -0.5109292268753052, -0.3147372305393219, 1.29181969165802, -1.7229961156845093, 1.5951502323150635, -0.4430321156978607, -1.270822525024414, -0.6242398023605347, 0.5568051934242249, -1.5034302473068237, 0.03537273034453392, 1.8672256469726562, 0.2654506266117096, -1.0129464864730835, -1.1941609382629395, 0.26928195357322693, -2.0067296028137207, -0.15119804441928864, -1.4842605590820312, -0.9621415734291077, 0.0823879987001419, 0.13746967911720276, -0.1640072911977768, 0.01252936664968729, -0.5675477385520935, 1.0091745853424072, 0.7347785830497742, -1.6664382219314575, 1.2174791097640991, 0.25124168395996094, -0.2281983643770218, -0.03303563967347145, 1.7865575551986694, -0.0950457751750946, -0.051381442695856094, -1.6903611421585083, 1.1242880821228027, -1.806908369064331, 0.4980422258377075, -1.118900179862976, -0.09402666240930557, -0.47933730483055115, 0.05274367332458496, 1.238836407661438, 0.30738410353660583, -0.06996043771505356, -0.0030333874747157097, -0.0471990630030632, -0.873883068561554, -0.3207443058490753, 0.47571468353271484, -1.0814404487609863, 0.3223167359828949, 1.686490535736084, 1.5762712955474854, -0.848261833190918, 0.5240224599838257, 0.47329458594322205, 1.2827914953231812, -0.16958028078079224, 3.7438786029815674, -0.1726628988981247, -0.014591085724532604, -0.4737691283226013, 0.5867561101913452, 0.737886369228363, 0.050966497510671616, 1.0120683908462524, 0.5168468356132507, -0.952394962310791, 0.591572642326355, -0.3470631539821625, -0.7911478877067566, 2.371169090270996, 0.5198060274124146, 0.389081209897995, 1.4591190814971924, -0.8812312483787537, 0.717501163482666, 1.104032278060913, -2.592008786450606e-05, 1.211316704750061, 1.0897809267044067, 1.3959752321243286, -1.3661121129989624, 0.3094066381454468, 0.9577026963233948, -1.7109426259994507, 0.8071890473365784, -0.008810913190245628, -1.0496159791946411, -1.5383479595184326, -0.7255808115005493, 1.5053011178970337, 0.6998923420906067, 0.46680137515068054, -0.976385235786438, -0.7633032202720642, -0.3455658555030823, -0.30182573199272156, -1.1385393142700195, -0.4318816363811493, 1.939078450202942, -0.06159703806042671, -0.1508890986442566, 0.905478298664093, 0.20336194336414337, -1.2676355838775635, -0.2829301059246063, -0.18616516888141632, 0.8779808282852173, 0.7771826386451721, 0.8020845055580139, 0.6589508056640625, -1.2049217224121094, 0.7782658934593201, 0.7151218056678772, -0.8386407494544983, 1.029396891593933, 0.6478089690208435, -1.6156837940216064, -0.04147212207317352, 0.2691411077976227, 1.8769866228103638, 0.4387390613555908, -1.2868930101394653, 0.45882779359817505, -0.7847689390182495, 0.8115856051445007, 1.5906121730804443, -0.398380845785141, -0.31417155265808105, 0.21818743646144867, -0.4672432839870453, 0.5282741785049438, -0.45000436902046204, 1.731925368309021, 0.7122933864593506, -0.676636278629303, 0.3364461660385132, -0.7536689639091492, -1.3185757398605347, -0.7459210157394409, 0.6564573645591736, -1.2649104595184326, 0.5046184659004211, 1.5572577714920044, 0.03573547303676605, -1.3649957180023193, 0.22223982214927673, -1.675302267074585, 0.2148154228925705, 0.4250671863555908, 1.052378535270691, 1.6204637289047241, 0.6668338179588318, -0.9642476439476013, 0.25631290674209595, 1.4325087070465088, 0.9776706695556641, -0.5156185626983643, 0.8376352190971375, 0.7092006206512451, 1.4789230823516846, 0.8520383238792419, 0.5988869071006775, -1.1060402393341064, -1.8443034887313843, -0.5307489633560181, 0.08613339066505432, 0.04760708287358284, -0.12339341640472412, 1.671949863433838, -1.3248655796051025, -0.2507619857788086, -0.44362157583236694, 3.2210793495178223, 0.37131398916244507, -0.3203451335430145, 0.317414790391922, -0.5667387843132019, -0.582662045955658, 0.82253497838974, 0.7178060412406921, -1.9324676990509033, -2.0274295806884766, 1.4414397478103638, -0.7399153113365173, 0.1382443606853485, -0.41011130809783936, 1.0984545946121216, 0.08907777816057205, 0.9624313712120056, -0.3490664064884186, 0.08198108524084091, -0.4382076561450958, 0.3503062129020691, -0.9733350872993469, 0.7893692851066589, 0.5987419486045837, 0.34921371936798096, 1.3176602125167847, 0.26925286650657654, -1.4007021188735962, -0.4149332046508789, -0.6099559664726257, 0.015995290130376816, -0.5002937912940979, -0.26859837770462036, -0.7447707653045654, -1.7799521684646606, -0.023483596742153168, -0.7063077688217163, -0.0645490437746048, -0.7922185063362122, -0.6858502626419067, -1.6715408563613892, -1.3085250854492188, 0.02887018956243992, 0.4161597490310669, -0.42794689536094666, 0.6196994185447693, 0.22505521774291992, -1.2526531219482422, 0.5235443115234375, 0.1457127034664154, -0.21102017164230347, 1.40677809715271, -1.412906527519226, -1.2122067213058472, 0.8335821032524109, -0.17249265313148499, 0.1410008668899536, -0.3594859540462494, 0.6067954897880554, -0.40767160058021545, 1.200900673866272, 0.08231214433908463, -0.7951557636260986, -0.35974085330963135, -0.9967216849327087, 0.7154279351234436, -0.0703105553984642, -1.6213891506195068, -1.8494516611099243, 0.7128239274024963, 0.2747749984264374, 0.1529223620891571, -0.004436492919921875, -0.7267215251922607, -2.2650070190429688, -0.3235994875431061, 1.132663607597351, -1.1430577039718628, 0.05464346706867218, -1.4193164110183716, 0.495846688747406, 0.5582399964332581, 0.38297218084335327, 0.18323324620723724, -0.1251860111951828, 1.3373767137527466, -2.764004707336426, -1.069614291191101, -0.9728304743766785, -0.9702010154724121, 0.9935269355773926, 0.9989650249481201, -1.6133965253829956, -0.31985852122306824, -0.2189049869775772, 1.194493293762207, -0.9181699156761169, -1.5790083408355713, 1.3439602851867676, -0.19683624804019928, -1.1183545589447021, 0.6050546169281006, -1.3253138065338135, -1.5655319690704346, 1.023077130317688, 0.4262247085571289, 0.8827160000801086, 0.05941774696111679, 0.816744327545166, 1.0733062028884888, -0.09642960876226425, -0.506432831287384, 1.0859001874923706, -0.21348656713962555, 0.5185139775276184, 1.0876591205596924, 1.0845448970794678, -0.2635909914970398, -0.9694153666496277, -0.7642941474914551, 0.6395027041435242, -1.1847032308578491, 0.9300029277801514, 1.8050166368484497, -0.4965018033981323, -1.3461809158325195, -1.9587793350219727, 1.0471251010894775, -1.51768159866333, 0.3336254358291626, 0.4725280702114105, 1.2618931531906128, 1.119307518005371, -0.5777609944343567, 0.6887668371200562, 1.4127354621887207, -1.1288954019546509, -0.10379872471094131, -0.7666208744049072, 0.08061366528272629, 0.6980980634689331, -1.9691740274429321, -0.18028108775615692, 0.20571930706501007, -0.7164198756217957, 0.4519243836402893, -0.22467659413814545, -0.48006871342658997, 0.4334971606731415, -0.7496842741966248, 0.6832166314125061, -0.1466888040304184, -0.4248426854610443, -0.7783799767494202, 1.1160144805908203, 0.2902159094810486, 0.4991730749607086, -1.1592109203338623, -0.21397005021572113, 0.8622171878814697, 0.12837204337120056, 0.7176682949066162, -1.3662792444229126, 1.439328908920288, -0.3629136383533478, -0.3101233243942261, 0.2278561294078827, 1.4372860193252563, -0.6508391499519348, 0.26725900173187256, -1.0797194242477417, 1.3967702388763428, -0.18920928239822388, -0.8197191953659058, -0.9676376581192017, 0.8270608186721802, -1.2270604372024536, 0.507731020450592, 1.3604072332382202, -2.099543333053589, -0.15417349338531494, 0.5734434723854065, 0.47415685653686523, -0.3131878077983856, -0.14463643729686737, 2.035048007965088, -0.3697408437728882, -0.18068133294582367, 1.3022544384002686, -0.7317513823509216, -0.46134305000305176, 0.46669283509254456, -0.22335445880889893, -1.165480613708496, 0.6743085980415344, 0.571110188961029, 0.7206911444664001, -0.570697546005249, 0.22092370688915253, 0.7393481731414795, -0.9776551723480225, 0.10646988451480865, 0.4129399061203003, 0.5965061783790588, -0.04288512095808983, -3.174691677093506, -2.433311700820923, -2.041149139404297, 0.35204604268074036, 0.3904971182346344, 1.3807365894317627, -0.9151438474655151, 0.9166362881660461, 0.6181626915931702, 0.35814139246940613, -0.7851047515869141, 0.18044136464595795, -0.6487338542938232, 0.7436431646347046, -2.745450496673584, 0.5267558693885803, -0.503696084022522, 1.500158429145813, 0.154195174574852, 0.1869230419397354, 0.6750701069831848, 1.239842176437378, 0.7201595902442932, -1.2139791250228882, -1.0362695455551147, 0.6420475244522095, 0.17514729499816895, -0.4807835817337036, 1.195081114768982, 0.9321243166923523, -0.7258733510971069, -0.61716228723526, 1.0855354070663452, 0.8673739433288574, -0.4689834415912628, -0.030987177044153214, -0.013540803454816341, 0.39794570207595825, -1.2072808742523193, 0.11180194467306137, -0.6609201431274414, 1.6753554344177246, -1.0787714719772339, 1.784681797027588, -1.8393670320510864, 1.5706396102905273, -0.8474122285842896, -0.22378495335578918, -0.5905855298042297, 2.432969093322754, -0.48120200634002686, 1.4644404649734497, 0.4046209156513214, 0.34116286039352417, -0.9501427412033081, 0.34834668040275574, 0.04960145428776741, -0.8384824395179749, -0.45614534616470337, -1.3526525497436523, -1.1260106563568115, 0.32338088750839233, 1.0465384721755981, -0.500606894493103, 1.0545434951782227, 0.012087772600352764, 1.440744400024414, 1.1848503351211548, 1.167751669883728, 0.14267286658287048, -1.1943259239196777, 0.7411754727363586, 0.7145435214042664, -0.40522581338882446, 1.2978098392486572, -1.6982980966567993, 0.4292333424091339, 0.3996681571006775, -1.6594798564910889, -1.3936570882797241, -0.9550072550773621, -0.9621948003768921, 1.6014246940612793, 0.7510422468185425, -0.1743139624595642, 0.8526308536529541, 0.47142818570137024, -1.3183565139770508, -1.610849380493164, 0.15671643614768982, 0.818520188331604, 0.10414806008338928, 1.6031451225280762, 1.191603183746338, -0.6557736992835999, 1.1678991317749023, -0.4285859763622284, -0.688273549079895, 0.7856956720352173, 0.3744463324546814, 0.9289640188217163, -0.04383542761206627, -1.3532640933990479, 0.955615758895874, 1.3716282844543457, -0.04546389356255531, 0.684485137462616, -2.204402446746826, -0.766928493976593, 0.7512661218643188, 0.4822056293487549, -0.15078221261501312, 1.572386384010315, -0.4590536057949066, -1.1196895837783813, -0.5955915451049805, -0.9990716576576233, 0.4667509198188782, 0.8175287246704102, 0.8822222352027893, -0.881361186504364, 0.24511423707008362, -0.9137349724769592, 0.6294090151786804, 0.5015697479248047, -0.8651090264320374, -0.5484043955802917, -0.5372628569602966, -0.1456301510334015, -1.234026551246643, 1.3037834167480469, -0.6000134944915771, 0.7187923789024353, -0.22797848284244537, 0.2713823616504669, -0.26642706990242004, -1.7611867189407349, 0.5199143886566162, 1.1803622245788574, 3.153211832046509, -0.6475535035133362, 0.7969393730163574, 0.11754520237445831, 0.5609633922576904, -0.3756682872772217, 0.8724482655525208, 1.6127889156341553, 1.1600184440612793, 0.06592916697263718, 0.04569732025265694, 1.2937932014465332, -0.4488726556301117, -2.3038723468780518, -2.093862533569336, -0.9326248168945312, 1.4146921634674072, 1.171370267868042, 0.32562994956970215, -0.7231050729751587, 1.4375194311141968, -0.15520107746124268, -0.6624829769134521, -1.7317663431167603, 0.6608782410621643, -0.8626242876052856, -0.02783137559890747, -1.7692385911941528, -1.269960880279541, -0.3284110128879547, 0.1549289971590042, 0.3607046902179718, 1.463626742362976, 1.4739729166030884, 0.4051207900047302, 0.060379281640052795, -1.0779372453689575, -0.9005875587463379, 0.429805725812912, -0.4068940281867981, -0.6837447881698608, 0.66085284948349, -0.6213604211807251, -1.176668643951416, 0.12748071551322937, 1.2777628898620605, -1.082911729812622, -0.25869622826576233, 0.5595201849937439, -0.8262091875076294, 0.3930444121360779, 2.214475154876709, 0.8645656108856201, -1.9294601678848267, -1.2782220840454102, 1.526665210723877, 2.8386669158935547, 0.7791415452957153, 0.3764704167842865, 1.4390084743499756, -0.4118407070636749, -0.5845587253570557, -1.1193408966064453, -1.5251444578170776, -0.9121578931808472, 0.53604656457901, 0.09003692865371704, 0.7477673292160034, 0.736147940158844, 1.9737780094146729, 1.1103534698486328, 0.4881592094898224, 0.5378618836402893, 0.9681466221809387, -1.0334515571594238, -0.051785893738269806, 1.1618256568908691, -0.2040763646364212, -0.7215628623962402, -2.2593441009521484, -0.16784778237342834, -1.2377643585205078, 0.09473808854818344, 1.0357065200805664, 0.13381327688694, 0.44706159830093384, -0.819176435470581, 0.7897241115570068, -1.338927149772644, 0.07821772992610931, 1.3363449573516846, -0.4752542972564697, 0.5827319622039795, -0.03268659487366676, -0.045826975256204605, -0.5297694802284241, -0.43869778513908386, 1.6239560842514038, 1.0379395484924316, -0.01180603913962841, -0.46419185400009155, 1.9055249691009521, -1.4148672819137573, -0.24460142850875854, -0.11812938749790192, -0.8689404129981995, 0.053930941969156265, 1.3781442642211914, -2.235095500946045, -0.6079740524291992, 0.07142118364572525, -1.0902202129364014, 0.14416055381298065, 0.29688698053359985, -0.4089679419994354, -0.6937894225120544, -0.4528757631778717, 0.047012876719236374, -0.2250414490699768, -0.5858087539672852, 0.7795723080635071, -1.8497037887573242, 2.3648738861083984, -1.4851958751678467, 1.4534389972686768, 1.4830635786056519, 0.1973903775215149, -1.0170347690582275, 2.1537082195281982, -1.378653645515442, -0.7721429467201233, -0.4514109194278717, 0.6342620253562927, 2.343303918838501, -0.11194173991680145, 1.975096344947815, 0.76689213514328, 1.5958404541015625, 0.889378011226654, 2.2785181999206543, -0.8000775575637817, -0.18205490708351135, 0.5748779773712158, 0.592393159866333, 1.2921432256698608, -0.4600982367992401, 0.06285443156957626, -0.649789571762085, -1.1515969038009644, 0.5801634192466736, -0.8978906869888306, -0.41864198446273804, -1.3802454471588135, -0.16017261147499084, -0.4846075475215912, 0.6469534039497375, 2.242156744003296, -1.926321268081665, -1.5049289464950562, -0.273807168006897, -1.415518879890442, -0.7593748569488525, 1.2363249063491821, -1.6121349334716797, -0.1911003291606903, -0.4387655556201935, 0.46402183175086975, -0.9789329767227173, 1.004470705986023, -0.1412939429283142, 0.43153488636016846, 0.27383893728256226, 0.009388295002281666, -0.701223611831665, 1.1346569061279297, 0.3791472017765045, -0.16472001373767853, 0.6889520883560181, 0.025675896555185318, 0.7607961893081665, -1.4648547172546387, -0.02743990533053875, -0.3386467397212982, 1.0016695261001587, 0.46422189474105835, 0.9726600646972656, 0.658037543296814, -1.1752238273620605, 0.09714489430189133, -0.6795850396156311, -0.05015010014176369, 0.42605772614479065, -0.9150471687316895, -0.8285112977027893, 0.2218092679977417, 0.8966567516326904, -0.7318956255912781, -0.12202397733926773, 0.9582690596580505, -1.3029054403305054, -0.8096752166748047, -1.2546639442443848, -0.982625424861908, 1.4223958253860474, 2.237307071685791, -0.728463351726532, -0.4683631658554077, -0.5469495058059692, -1.291630506515503, -1.6257178783416748, -0.43832603096961975, -1.1750653982162476, 0.8056580424308777, -1.0865318775177002, 0.7016377449035645, 0.7831676006317139, -1.2151203155517578, 1.1210675239562988, 0.2774181067943573, 1.222123146057129, -0.2890946865081787, -0.590090274810791, -0.8245130181312561, -0.7862061262130737, 0.5272668600082397, 2.315835475921631, -0.17794129252433777, 1.3803926706314087, -0.7352577447891235, 0.6643505692481995, -0.9687851667404175, -0.9055399894714355, -2.4820008277893066, -0.0265404861420393, 0.9178462624549866, -0.1632029414176941, -0.6102824211120605, -0.16872890293598175, -0.6886029243469238, -1.1590518951416016, 0.01255959365516901, -0.9539698958396912, -0.5979876518249512, -2.0355026721954346, 0.30912521481513977, -0.012032832019031048, 0.6120135188102722, -0.1321500688791275, -1.4010556936264038, -1.1272788047790527, -0.6877890229225159, -0.3028270900249481, -0.13436049222946167, 0.9226884841918945, -0.3796040415763855, 0.2679443359375, 0.9926201105117798, -0.9707216024398804, 0.06954894959926605, 0.343356192111969, -1.0556727647781372, 1.6956533193588257, 1.1253160238265991, -0.4665035009384155, 0.6028942465782166, -2.2419028282165527, 0.8401477336883545, 0.8618381023406982, 1.5318275690078735, -0.1527981013059616, -0.2373579442501068, 1.0771974325180054, -1.2173124551773071, 0.154537633061409, 0.2538827657699585, -0.09263745695352554, 0.3023681640625, -0.08206202834844589, 1.6566591262817383, -0.6989302635192871, -1.1507230997085571, -0.7710728049278259, -1.2788100242614746, 1.3823456764221191, -0.6867339015007019, -0.43017256259918213, -0.05073997750878334, 0.19768375158309937, 1.9788455963134766, -0.6861169338226318, 0.08925245702266693, 0.580810010433197, 0.14941057562828064, 1.351783275604248, 1.2650933265686035, -0.702370285987854, -1.2547507286071777, -1.1670886278152466, -0.6176609992980957, -1.0203839540481567, 0.22005431354045868, -0.5886659026145935, 0.28799954056739807, 0.19791869819164276, -0.46206170320510864, 1.593364953994751, 1.157311201095581, 0.5868714451789856, 0.3097522556781769, -1.0849246978759766, 0.963760495185852, -2.2565083503723145, 0.7020094394683838, -1.2259808778762817, -0.3749150335788727, -0.6891211271286011, 1.266025424003601, 1.969841718673706, 2.025144100189209, -0.7835463285446167, 0.9666841626167297, -0.722885251045227, 0.9446549415588379, 1.1833914518356323, 1.514047622680664, -1.1515052318572998, 1.0118387937545776, 0.003792880102992058, -0.010137725621461868, -0.3096245229244232, 0.37301406264305115, -1.125927209854126, 0.016951890662312508, -0.36488839983940125, 0.6287105679512024, -0.1397947371006012, 0.5342733860015869, -0.6631160974502563, -1.046608567237854, -0.21808740496635437, 1.0509657859802246, 0.9526600241661072, -1.9954420328140259, 0.31050699949264526, 0.25700831413269043, 1.1514506340026855, 0.23468905687332153, -0.4472845196723938, 0.3928094208240509, 0.5153622031211853, -0.6681888103485107, -0.03676408901810646, -1.151824951171875, 0.7642726302146912, 0.04158446192741394, 0.6913028955459595, -1.516594648361206, 0.7649035453796387, -0.042580753564834595, 0.666797399520874, -0.4269348680973053, 1.4379713535308838, 0.4693600833415985, -0.24555791914463043, -0.16976012289524078, 0.13376690447330475, 1.8064837455749512, 1.5337791442871094, 0.10131603479385376, 0.5781402587890625, -0.6823870539665222, 0.18541045486927032, 1.7754132747650146, -0.4248817265033722, -0.6877768039703369, 2.536752223968506, 0.08963089436292648, 2.108999490737915, 0.9248070120811462, -0.16810165345668793, -1.95441734790802, 1.2594481706619263, -0.4036886990070343, -1.4974294900894165, 0.16146790981292725, 0.6415865421295166, -1.2081634998321533, -0.9576208591461182, 0.37211331725120544, 1.3340139389038086, 1.3995692729949951, 0.35034462809562683, 0.2773262858390808, 1.201092004776001, 0.3050389289855957, -0.151678204536438, 2.4856419563293457, 0.13370153307914734, -0.8655529022216797, -0.10079671442508698, 0.07318545132875443, 0.4373686611652374, 0.4652814567089081, -0.7244836688041687, -0.6428727507591248, -1.8928533792495728, -0.8851339221000671, -0.9423004388809204, -0.8280285596847534, 1.5967849493026733, -1.5362803936004639, 0.36988383531570435, -0.6419773697853088, -0.010447611100971699, -0.44828253984451294, 1.401296615600586, -0.5711353421211243, 0.13395856320858002, -0.41019365191459656, 2.3999178409576416, 1.1362718343734741, 0.6903945803642273, -0.28803521394729614, -0.7836143374443054, -0.27000218629837036, -0.17215591669082642, 0.26421213150024414, 0.6078048348426819, -0.9138469696044922, -0.1554928869009018, 0.23728559911251068, 0.4402054250240326, 0.7759833335876465, -0.672710657119751, -2.0222465991973877, 0.470639169216156, 0.04441507160663605, 0.6718868613243103, -0.7079305052757263, -0.26236602663993835, -1.8286932706832886, -0.6440522074699402, -1.5495808124542236, -0.2964879274368286, -1.4850510358810425, 1.2202445268630981, 0.12880542874336243, 1.3274763822555542, 1.5063672065734863, -0.5024469494819641, 0.15800823271274567, 0.533474326133728, -0.269548624753952, -0.32985347509384155, -0.8636651635169983, -1.4728907346725464, -0.4640316963195801, -0.6958651542663574, 0.9555736780166626, 1.8624377250671387, 0.19561956822872162, -1.6252628564834595, -1.177841305732727, -0.37960803508758545, 0.29227226972579956, 1.0132921934127808, 1.3256702423095703, 0.4182584583759308, -0.5157049894332886, 0.796339213848114, -0.13589385151863098, -1.4511497020721436, 2.381810426712036, -0.08991966396570206, -0.2991882264614105, -0.5975213050842285, 0.7388677597045898, -0.26152902841567993, 1.3434520959854126, -0.5336725115776062, -1.025895118713379, 1.426589846611023, 0.47323280572891235, -1.4325449466705322, 0.7887610793113708, 1.1743285655975342, -0.8330200910568237, 1.8170257806777954, -0.06991277635097504, -0.27385202050209045, 0.930618166923523, 0.2907320559024811, -0.17264555394649506, 0.5844979286193848, -2.190363645553589, 1.5806232690811157, -1.8048174381256104, 0.19845812022686005, -0.4105512201786041, 0.7280724048614502, -0.38671985268592834, 2.659893751144409, 1.2052838802337646, 1.9463913440704346, 0.586584746837616, -1.8852893114089966, 0.323703408241272, -0.34295302629470825, 0.6281999945640564, 0.6561405658721924, -0.12590815126895905, -0.8644442558288574, 0.04347284883260727, 0.3814541697502136, -0.548617422580719, 0.6338139176368713, -0.6344155073165894, 0.30905774235725403, 0.2744014859199524, 0.21337467432022095, 1.4629907608032227, 1.0948818922042847, 0.1298922449350357, -0.4200912415981293, 1.2197012901306152, 0.34295472502708435, -0.9602785706520081, -0.7144013047218323, 1.1154892444610596, 1.2083632946014404, 0.25877195596694946, 0.21908317506313324, 1.252317190170288, 1.8089629411697388, -1.4130853414535522, 1.2899072170257568, -0.8952476382255554, -0.25080037117004395, 0.6484904289245605, -0.9464783668518066, 0.9374585151672363, 1.4761943817138672, 0.9596258997917175, -1.1554805040359497, 0.031066838651895523, -0.11526361107826233, 1.2668601274490356, -0.4070810079574585, 3.033971071243286, 0.477570503950119, 0.09414354711771011, 0.1986808031797409, -0.024800365790724754, -0.5240570902824402, 0.42412668466567993, -0.389614075422287, 1.0370372533798218, -2.503438949584961, -0.021381808444857597, 0.836094319820404, 1.482009768486023, 0.16212747991085052, -0.641925573348999, -0.1093255952000618, -1.6412924528121948, 2.055119276046753, 0.43279609084129333, 1.9314302206039429, -0.733819842338562, 0.6579167246818542, -0.6920340061187744, -0.021462364122271538, -0.9106907248497009, -0.15150535106658936, 1.6595771312713623, -0.01028874609619379, -0.7861226797103882, 0.1721692979335785, 0.43304184079170227, 0.7337529063224792, 0.9747509956359863, 0.9147266745567322, 1.8277839422225952, -0.36889728903770447, 0.43121829628944397, -0.56029212474823, -0.5090141892433167, -1.6171244382858276, 0.2719837725162506, 0.08187758922576904, 1.687252402305603, -0.13785910606384277, -0.10346641391515732, -0.46547597646713257, -1.43926203250885, 1.6246825456619263, -0.2374611645936966, 0.06237515062093735, 0.49750691652297974, 0.09704115986824036, 0.4499516487121582, 1.2917215824127197, 0.5245510339736938, 0.390167236328125, -0.6870587468147278, -1.4799524545669556, -0.8956151604652405, -0.5389144420623779, -0.8536889553070068, -0.8848636150360107, 1.6965614557266235, -1.1495863199234009, -0.5055264234542847, 0.6915889978408813, -0.49041640758514404, 0.828002393245697, -1.9181841611862183, 0.25664061307907104, -0.8811922669410706, 1.2265100479125977, -0.9392132759094238, -1.5897138118743896, 0.17002369463443756, -0.4235181212425232, 0.587882936000824, -0.5168603658676147, 2.5412161350250244, 0.3210376501083374, -0.3462354242801666, 0.17499187588691711, 0.7621676325798035, 0.29018816351890564, 0.9065998792648315, 0.3706481158733368, -1.7514960765838623, 1.5687401294708252, 0.10401720553636551, 0.46877244114875793, 0.40468743443489075, -0.8590096831321716, 0.029595257714390755, 0.7202211022377014, 1.967527985572815, 0.8955660462379456, 0.06425649672746658, 1.0469812154769897, 1.417191505432129, 0.32280534505844116, -0.31232231855392456, 2.753866195678711, -0.251182496547699, -0.6454158425331116, 0.33448681235313416, -0.2682918906211853, 0.6047282814979553, -1.4847862720489502, 1.358644723892212, -1.442291021347046, -0.8185243606567383, 0.38153383135795593, 0.08869174867868423, 0.5057545900344849, 0.280735582113266, -0.06598016619682312, -1.5959059000015259, -1.0047763586044312, -1.1024534702301025, 0.5922526717185974, -0.9517098665237427, -1.0625543594360352, -2.7647340297698975, -0.5357697606086731, 0.4067976176738739, 0.950676679611206, -0.3743414580821991, 0.6119585633277893, -0.04554382339119911, 0.4240347146987915, -1.3511236906051636, -0.6179496645927429, -0.5384029150009155, -1.2002732753753662, -0.4948640465736389, 0.21239201724529266, 0.09966596961021423, -0.0907663106918335, 1.710781216621399, -1.184802770614624, 0.7946518659591675, 0.21895048022270203, -0.7696505784988403, 0.6144176721572876, 2.0328385829925537, -1.9715383052825928, 2.522299289703369, -0.407720685005188, -0.39221516251564026, -0.6596804261207581, -0.7339963912963867, -2.02410888671875, -0.17321106791496277, 1.5200434923171997, -0.4675760865211487, -0.6096993684768677, -0.6181526184082031, 0.3382001221179962, -1.5358854532241821, 0.6688875555992126, -0.6982647180557251, 0.9613651633262634, 0.044095009565353394, -1.962090253829956, 1.7357347011566162, -0.3191797733306885, -0.141629159450531, 1.0404317378997803, 0.9981854557991028, 1.3780373334884644, 0.11477743089199066, 0.9280248284339905, -0.3584069311618805, 0.16234345734119415, 0.43849533796310425, 0.9534696340560913, -1.5646650791168213, 1.1424471139907837, -1.1228984594345093, -0.3483096659183502, -0.36146488785743713, 1.6191892623901367, 1.7764509916305542, 0.820594072341919, 0.19154639542102814, 0.5036064386367798, -0.8736037611961365, -1.2634769678115845, -0.7859111428260803, 1.7804062366485596, 0.22885674238204956, 0.011452103964984417, -0.025223534554243088, 1.1644928455352783, 0.7737460136413574, 0.36326658725738525, -1.5261027812957764, 0.4314727187156677, 0.18023401498794556, -0.6018730401992798, 1.4820765256881714, -1.2446322441101074, 0.6341810822486877, -1.1327965259552002, 0.3535526394844055, -0.33447131514549255, -1.7119337320327759, 0.3435226380825043, 0.39004620909690857, 1.1196264028549194, 0.26731082797050476, 0.8179733157157898, 0.7012505531311035, -0.4737808108329773, 1.4695714712142944, 1.1092561483383179, 1.4560810327529907, 0.19261056184768677, 1.7774924039840698, -1.9988378286361694, -0.8550533056259155, -0.32869067788124084, 1.8948028087615967, 0.8432823419570923, 1.6151942014694214, -0.8461720943450928, 0.10186862200498581], [-1.3409569263458252, -0.10454610735177994, -0.6217904686927795, 1.2740966081619263, 0.17851902544498444, 0.10156231373548508, 2.13118314743042, -0.7210261821746826, -1.0057576894760132, -0.30985549092292786, 0.1412380486726761, 1.227689266204834, 2.863297939300537, -0.07042202353477478, -1.9190021753311157, 0.9936510920524597, -0.8400089144706726, 1.111687183380127, -0.6135331988334656, -0.946864664554596, 0.6067636609077454, -1.1956526041030884, -0.5017492771148682, -0.6119850873947144, -0.8189643025398254, 0.6485742926597595, 0.25954529643058777, 1.2865651845932007, 2.0966176986694336, -0.2550947666168213, -1.0333491563796997, 0.1427788883447647, -0.6241902112960815, 0.006442222744226456, -0.007390910759568214, 1.1171237230300903, 0.3763286769390106, -1.421315312385559, -0.3216806650161743, 0.11992587149143219, 0.897292971611023, -2.148239850997925, -0.8940156698226929, -0.7241185903549194, 0.07419758290052414, -0.28952014446258545, -0.3433694541454315, 0.4154700040817261, 2.23482346534729, -0.7704529762268066, -2.962419033050537, 0.5958145260810852, -0.591411292552948, -1.496313214302063, -0.39043116569519043, 0.3145468831062317, -0.47000396251678467, -0.4472872018814087, -0.527391791343689, -0.8299132585525513, 3.1653149127960205, 0.757208526134491, 1.2936304807662964, 1.6210216283798218, 0.4176345467567444, 0.44544681906700134, -1.8646169900894165, 0.29349443316459656, 0.1626071035861969, 1.545924186706543, -2.5205471515655518, 0.586452066898346, -0.42307332158088684, -1.6706304550170898, -0.03720012679696083, 0.7386562824249268, -0.6164060235023499, -0.598358154296875, -0.9098535180091858, 1.5193698406219482, 0.3239779472351074, -0.46317052841186523, 0.633023738861084, 1.740204930305481, 1.0476665496826172, -0.3763936161994934, 0.017714884132146835, 1.4642333984375, 0.07815753668546677, 1.0576932430267334, -0.6237912774085999, 0.36644378304481506, 0.18137681484222412, 0.42358529567718506, 0.947279155254364, -0.8851318955421448, 2.3210971355438232, 0.457816481590271, -1.4765996932983398, -0.21701361238956451, -0.7103395462036133, -0.3122800290584564, 0.2159748524427414, 0.17376019060611725, 0.215193510055542, 0.7060468792915344, -1.7597931623458862, 1.1160582304000854, -0.5952239632606506, 0.5295711755752563, 0.19727085530757904, 0.9934853315353394, -0.9500434994697571, -0.7886180877685547, -0.00904262624680996, -0.16029532253742218, -1.6098971366882324, -1.5728721618652344, -0.5108664035797119, 0.618110716342926, 0.3027458190917969, 0.40459364652633667, 0.8870367407798767, -0.6913487315177917, 0.798438549041748, 1.5727660655975342, -0.54876708984375, -0.37707212567329407, 0.10100150108337402, -0.17448382079601288, -1.988749623298645, -0.18453851342201233, -1.1243728399276733, 0.6022950410842896, -0.06393083184957504, 0.23116125166416168, -0.631928026676178, -0.026980066671967506, -1.0981614589691162, -0.626720130443573, 0.032500408589839935, 1.6087560653686523, 0.1658536195755005, -1.269321084022522, 0.2827538251876831, 0.14616137742996216, -0.8569872975349426, -0.011155852116644382, -0.3096741735935211, 1.4399112462997437, 0.17264659702777863, 1.3972364664077759, -0.5788547396659851, -0.40657368302345276, -0.4118281602859497, 1.4002752304077148, 0.9554299712181091, 0.3755703270435333, 0.3530289828777313, -0.8527960777282715, -0.4084061086177826, -0.2559334337711334, -0.2349921017885208, -0.10022728145122528, 1.3318653106689453, 0.08005239814519882, 0.8705698251724243, -0.9836068153381348, -0.6290673613548279, -0.23090791702270508, 1.907559871673584, 0.9629133343696594, 0.8693094849586487, -0.9241988062858582, 0.4570959806442261, -0.7388978600502014, 0.5614221692085266, 0.9012295603752136, 1.9349467754364014, -0.26017799973487854, 0.7426950931549072, -0.6195374131202698, 0.3930892050266266, -0.663701057434082, -0.04780496284365654, -0.2996671497821808, -0.02391992323100567, -0.9122353792190552, -0.9821987748146057, -0.9572018384933472, -1.0317778587341309, 1.2219315767288208, 1.700849175453186, -2.21402907371521, -0.41103559732437134, -0.8559216856956482, 0.7395929098129272, -1.398087739944458, 2.1214191913604736, 1.2699099779129028, -1.1682158708572388, -1.348724365234375, 0.30297812819480896, 0.5421979427337646, -1.8519734144210815, -0.11306097358465195, -0.1459040492773056, -0.9374257922172546, 0.009580307640135288, -0.8725488185882568, 0.03460664302110672, -0.02058194950222969, 0.037652693688869476, 0.1431216150522232, -1.024938702583313, 0.1942177265882492, 0.8753582239151001, -0.0665811225771904, -0.029687274247407913, 0.29115623235702515, -0.6893362998962402, 0.7765698432922363, -1.7180598974227905, 0.011130930855870247, 0.564457893371582, 0.06986329704523087, -1.0142512321472168, -1.5217199325561523, 0.03940153867006302, -1.8577837944030762, -0.05918511003255844, -0.2386653870344162, -0.32558009028434753, -0.4664327800273895, 0.49893829226493835, 0.17209625244140625, -0.020987072959542274, 1.2628809213638306, -0.3316030502319336, -1.5373862981796265, -0.23216845095157623, -0.39541128277778625, -0.3369634449481964, -0.4760245680809021, -0.25591424107551575, -0.2794193923473358, 0.5105109810829163, -0.7071621417999268, -0.2631857693195343, 0.8570037484169006, -0.8388953804969788, 0.2698935568332672, -0.021261202171444893, 0.4551902711391449, -0.6779001951217651, 1.139313817024231, 0.6032202243804932, -2.484440565109253, -0.9939940571784973, 0.47882816195487976, -0.34415680170059204, -0.8306706547737122, -1.1568841934204102, 1.1382311582565308, -1.2894935607910156, 0.5516631603240967, -0.5562000274658203, -1.3991552591323853, 0.34697914123535156, -0.35303306579589844, 0.7940780520439148, -1.3254269361495972, -1.1322665214538574, -0.7294890880584717, 0.9571706652641296, -0.9419868588447571, 2.581144332885742, 0.8134626150131226, -0.5438669323921204, -1.475528597831726, 0.9983254671096802, 0.5794734954833984, 1.2077839374542236, -0.1915171891450882, 0.6729570031166077, -0.5054240822792053, -0.15729880332946777, -0.6236914992332458, -2.855698823928833, 0.5732942819595337, 1.1773566007614136, 1.645805835723877, 0.6530832052230835, 0.04489818960428238, -1.2114938497543335, 1.9550367593765259, -0.3831171989440918, 0.33515864610671997, 0.4677543044090271, -0.9652294516563416, -0.6587015986442566, 0.6084728837013245, 0.8543972373008728, 0.9131981730461121, 0.13773077726364136, -0.7881030440330505, 2.416930675506592, 0.9028981924057007, 0.5278859734535217, -0.14957784116268158, -0.4506584703922272, 0.33532455563545227, -1.1913079023361206, 1.3258447647094727, -1.269963026046753, 1.224619746208191, -0.39260751008987427, 1.0874077081680298, 1.1040241718292236, 0.44506996870040894, 2.021073579788208, 1.594738483428955, 0.6177436113357544, -0.9000937938690186, 0.11967586725950241, 0.49719128012657166, 1.0929349660873413, -0.20943593978881836, 1.1417579650878906, 0.665539026260376, 0.966676652431488, 0.3865037262439728, 1.916866660118103, 0.13009649515151978, -1.0662018060684204, -0.07348152250051498, 2.002027988433838, -1.4852961301803589, -1.878532886505127, -0.17865915596485138, -0.5806693434715271, -0.7880067825317383, 0.855027973651886, 0.25229302048683167, 0.4800390899181366, 0.21767260134220123, -0.1595257669687271, 0.004966862965375185, 0.5755566358566284, -1.566494345664978, 0.3259034752845764, 1.4581708908081055, -1.0729975700378418, 0.4916021227836609, -0.32441505789756775, 1.0968888998031616, -0.6249887347221375, -0.5352489948272705, 0.14674535393714905, -0.41795480251312256, 0.06497493386268616, 2.9473962783813477, -0.02832186222076416, 0.8148347735404968, -1.369109869003296, -0.47297874093055725, 0.9092802405357361, -0.2047412097454071, 0.1350000500679016, 0.8475995659828186, -0.16010242700576782, 1.039300560951233, 0.12442139536142349, 0.8088755011558533, 0.8118079304695129, 0.20931695401668549, -0.8035851120948792, 0.30714014172554016, -0.7580055594444275, 1.1197035312652588, 0.3156753480434418, -2.0998687744140625, 1.1457512378692627, -1.7733961343765259, -0.05762745067477226, 1.511512279510498, -0.4749866724014282, -1.2865314483642578, 0.7620315551757812, -0.5725680589675903, -0.3552974462509155, 0.280217707157135, -0.12483686208724976, 0.04147130623459816, -1.1482043266296387, 0.05989973247051239, 0.17667771875858307, -0.5268767476081848, 0.03680414706468582, -0.7179716229438782, 0.3240915536880493, 0.5839242935180664, 0.9184580445289612, -0.6656672358512878, -0.09879332780838013, 0.3989523947238922, -0.987951397895813, -0.4367075264453888, 1.235174536705017, 1.0821253061294556, -0.34556350111961365, 0.46448826789855957, 0.9329675436019897, -0.41724616289138794, 0.4900459349155426, 1.7883533239364624, 0.793095588684082, -0.9088271260261536, -0.6077104806900024, 0.07987578958272934, 0.22943882644176483, -0.4489721357822418, 0.3790663182735443, -0.005525441840291023, 0.523642361164093, -1.5984598398208618, -1.0941240787506104, 0.7413308024406433, 0.09094038605690002, 1.5329068899154663, -1.1333566904067993, -0.7078205347061157, 1.3371028900146484, 2.4232096672058105, -0.7155042886734009, 0.9646018743515015, 1.4418737888336182, -0.2637147605419159, 1.7455990314483643, 0.5897324085235596, -1.0256679058074951, -1.3725167512893677, -0.136687770485878, 0.5512160658836365, -0.8885830640792847, 1.2903509140014648, 0.8887104988098145, 0.3540246784687042, -0.17802663147449493, -0.003158753039315343, 0.4796721041202545, 1.3011634349822998, 0.16070207953453064, 0.45967885851860046, 0.4888705015182495, -1.7495646476745605, 0.4706096053123474, 1.3495314121246338, 0.5344915390014648, -0.3410387635231018, 0.8022577166557312, 0.3268885612487793, -0.12819314002990723, -0.550968587398529, -0.9868839383125305, 0.4990999400615692, 0.9559532999992371, 0.31536680459976196, -0.41954487562179565, -0.3301433026790619, -1.7985997200012207, -1.9640814065933228, 0.6398109197616577, 0.514197826385498, -1.1577478647232056, 1.0168046951293945, 0.889101505279541, -0.5091992020606995, -0.06221472844481468, 0.6974667906761169, -1.232201099395752, -0.6951950788497925, 0.7223902344703674, -0.373125821352005, -0.23645631968975067, -0.4869355261325836, -0.8973264694213867, -0.45006832480430603, -0.9948762655258179, -2.7932565212249756, -1.083567500114441, 0.20270273089408875, 0.3893418312072754, 2.167262077331543, -0.6286229491233826, -0.9561437964439392, 0.05746518820524216, -0.8233356475830078, 1.38791823387146, 0.9867919087409973, -0.5519164204597473, -1.2807443141937256, -0.40917810797691345, 1.2227075099945068, -0.02112732268869877, 0.4562170207500458, -1.2095156908035278, -1.3309638500213623, 0.8230248689651489, 0.6647846698760986, -0.7288491129875183, -1.5169414281845093, 1.0732455253601074, -0.6476975083351135, -0.11211878061294556, 0.8265678882598877, 0.8693535327911377, 2.089773416519165, 0.1824926733970642, -0.8173531889915466, -0.05102594196796417, 1.409812092781067, 1.2942878007888794, 1.411075472831726, 0.551130473613739, 1.5215598344802856, -1.7983851432800293, 0.5506261587142944, -1.1529972553253174, 0.49642878770828247, 1.3018897771835327, 1.1944760084152222, 0.19395309686660767, 0.396980881690979, -0.5789361596107483, -0.45495864748954773, 0.15490953624248505, -0.8358242511749268, 1.6471830606460571, 0.35307955741882324, -0.5128399729728699, -0.273791640996933, 1.4286444187164307, 0.5143975019454956, -0.8985868692398071, 0.37024229764938354, 0.6443290114402771, -0.40729162096977234, -0.46593108773231506, -0.8036074638366699, 1.2206735610961914, -0.46936506032943726, 0.4522680342197418, 1.3197542428970337, -0.7790340781211853, -0.1762775033712387, 0.9413943290710449, 0.44432327151298523, 2.9296679496765137, -0.44578874111175537, -1.1540790796279907, 0.19480252265930176, -0.03062496706843376, 1.1090788841247559, 0.3246897757053375, 1.079168438911438, -0.07845522463321686, 1.9584529399871826, -0.07241500169038773, 0.7795569896697998, -0.9948219060897827, -0.5805132985115051, 0.07911708950996399, 0.7068763375282288, -1.676645040512085, -1.5502451658248901, 0.8271868824958801, -0.10854151844978333, -0.9538737535476685, 0.377596378326416, -0.0706944465637207, -1.1956729888916016, 0.53858882188797, -0.5217997431755066, -1.5079498291015625, 0.07027889043092728, 0.8730124831199646, 1.1612225770950317, -0.239760160446167, -1.191053032875061, 2.097216844558716, -0.4028227627277374, 0.06700901687145233, 1.422486662864685, -1.2383860349655151, -0.30648165941238403, 0.26549866795539856, 0.012880872935056686, -1.1995184421539307, -0.1754457652568817, 1.7733995914459229, 0.012633171863853931, -1.733837366104126, -1.3335793018341064, -0.13841542601585388, 0.3857259154319763, 0.17037585377693176, -0.9115293622016907, 1.792350172996521, 1.8285139799118042, -0.6109827756881714, 0.5516909956932068, 1.3892897367477417, -0.1757558286190033, 0.5806596279144287, 0.4487890899181366, 1.2264454364776611, -0.36986595392227173, 0.5606002807617188, 1.4240453243255615, -0.7313118577003479, -0.5047351121902466, 0.7432802319526672, 0.34183964133262634, 0.5320019721984863, -1.6652418375015259, -1.185490369796753, 0.7737749814987183, -0.8716907501220703, -1.5835423469543457, -2.0342376232147217, 0.687204122543335, 0.6561256051063538, 2.3690035343170166, 0.7625954747200012, 0.4881683886051178, 0.23389537632465363, -0.8993706703186035, -0.3274996280670166, 0.9529334902763367, -0.23347212374210358, 0.23432999849319458, 0.3501420021057129, -1.1457475423812866, 0.27646979689598083, 0.6827059984207153, -0.00186934694647789, -0.15733765065670013, 0.6350504159927368, -2.272700309753418, 0.9246360063552856, 0.8902381658554077, 1.357622504234314, 1.2693226337432861, -2.0207653045654297, 0.540444016456604, 0.7057138681411743, -1.9572772979736328, -1.6281793117523193, 1.0684683322906494, 1.3521621227264404, -0.17183247208595276, 0.9352313280105591, -0.7918654084205627, 0.3225981891155243, -1.6548391580581665, 0.4484767019748688, -0.038424793630838394, 1.1008334159851074, -0.8827944993972778, -0.008080990053713322, -1.1123145818710327, 0.01969565451145172, -0.7842514514923096, -1.5982532501220703, 0.07425311207771301, 0.7047767639160156, -0.5424326658248901, -1.4158519506454468, -0.5031559467315674, -1.3161669969558716, -0.5261226892471313, 0.25153812766075134, 1.010391116142273, 0.19955390691757202, -0.5843397974967957, 0.1252693086862564, -1.931157112121582, -0.2477901428937912, 0.6840305328369141, -0.19804061949253082, 0.11015649139881134, -1.5121393203735352, -1.9194012880325317, -0.1751846969127655, 1.355863094329834, -0.8677572011947632, -0.6696308255195618, -1.5799005031585693, 1.3181637525558472, 0.41021955013275146, -0.15039975941181183, -2.4275965690612793, 0.24111077189445496, 0.41741275787353516, 0.8462098836898804, -0.3357456624507904, 1.7369389533996582, 1.5886645317077637, -0.1567164957523346, 0.40589067339897156, 1.1243208646774292, -1.2036092281341553, 2.0220839977264404, -0.5791987180709839, -0.43389028310775757, -0.29548174142837524, 0.40253832936286926, 0.442558616399765, 0.7355629205703735, -1.9314911365509033, -0.7685418128967285, -2.225095748901367, -3.01751971244812, 1.1770426034927368, -0.5014121532440186, -0.9444111585617065, -0.4113728702068329, -0.9088805317878723, -0.9396684169769287, 0.8526354432106018, 0.4117703437805176, 1.4035990238189697, -0.015138903632760048, -0.19745473563671112, -0.11249551922082901, -1.310154676437378, -0.710554838180542, 1.122279167175293, -0.16469404101371765, -1.4225177764892578, -2.1184849739074707, 1.2996408939361572, 0.8472772836685181, -1.6830865144729614, -0.3083561956882477, 0.9339380860328674, 0.9987863898277283, 0.4557940661907196, -0.43040454387664795, -1.2789591550827026, 2.1534969806671143, 2.069606304168701, 1.3474966287612915, -0.6510235667228699, 0.5651822686195374, -0.6963005065917969, 0.08654685318470001, 0.3075214624404907, 1.6880788803100586, -0.7865426540374756, -0.816578209400177, -0.5397220253944397, 0.6134780645370483, -1.1832407712936401, 0.007045861799269915, -1.0613422393798828, 0.6168561577796936, -0.24326366186141968, 0.8352784514427185, 1.0051933526992798, 0.4167090952396393, 1.765194058418274, 0.846161961555481, 2.567331314086914, 1.1836743354797363, -0.5438162088394165, -0.9563024044036865, 0.8636273145675659, 0.4917033910751343, -0.20687784254550934, -0.4374791085720062, 0.41641032695770264, 0.5127363204956055, -1.1333199739456177, 0.04922261834144592, -0.03682070970535278, -0.6768757700920105, -1.282058835029602, 1.5413116216659546, 1.2362415790557861, -1.337923288345337, -0.4460453391075134, -0.43179014325141907, -0.6575711965560913, 0.8257443308830261, -1.0063960552215576, -0.6520708203315735, -0.42012885212898254, 0.09263458847999573, 1.0239787101745605, 0.27330461144447327, 0.5627414584159851, -2.0233001708984375, -0.6139796376228333, 0.23559711873531342, 0.004504169803112745, -0.40385907888412476, -0.5782220363616943, -0.8200960755348206, -1.6483643054962158, -0.6090776324272156, 1.1220440864562988, 0.15322309732437134, -1.254654884338379, -0.4538312554359436, 0.16344542801380157, 1.7190481424331665, -1.8231327533721924, 0.028086945414543152, 0.19996482133865356, 0.08875343203544617, 1.6071757078170776, 0.19012515246868134, 1.6108555793762207, -0.12176002562046051, 1.325405478477478, 0.4741465151309967, -0.525566577911377, -0.1556437611579895, 0.5683707594871521, 0.8846935033798218, 0.35161644220352173, 0.4956469237804413, 1.354909896850586, -0.466774582862854, -1.7105696201324463, -0.06036154925823212, 1.0292422771453857, -0.1218191534280777, 0.26416030526161194, -0.7516375184059143, -0.3402586281299591, 0.013112714514136314, 1.663403034210205, 0.12043686956167221, 1.4052118062973022, -0.26258498430252075, -0.6671491861343384, 0.1643816977739334, 0.42851564288139343, 1.8169841766357422, -2.25795578956604, 0.38980361819267273, -1.1583094596862793, -2.014404535293579, 0.29149118065834045, -0.3520547151565552, -0.24777686595916748, -0.354573130607605, -1.3873268365859985, 1.33016836643219, 0.9179103970527649, 1.0248942375183105, 0.09745041280984879, -0.10407255589962006, 0.35168948769569397, 0.3810286223888397, 0.31293606758117676, -0.7745051383972168, 0.5864710807800293, 1.3385587930679321, -0.021381229162216187, 0.8009742498397827, 1.2371803522109985, -0.4514278769493103, -0.2560577690601349, 0.5633028149604797, -1.1798015832901, 1.3663954734802246, -0.42440563440322876, 0.3966112434864044, 0.0239980760961771, 0.1501110941171646, -0.1520174741744995, -0.2961041331291199, 0.8331301212310791, 0.24925927817821503, 0.5847976803779602, -0.6099728345870972, 1.2351834774017334, -0.9447481036186218, 0.49421441555023193, -1.1233924627304077, -0.048542287200689316, -1.4568201303482056, -0.23169845342636108, -0.3577762842178345, -1.2351531982421875, 0.5316439867019653, -0.14536233246326447, 0.9524708986282349, -0.013667299412190914, -0.27308863401412964, 0.9186947345733643, -1.7260262966156006, -0.6894248723983765, 1.1372536420822144, 0.46626630425453186, 0.4640049636363983, 0.2765665054321289, -0.11755666881799698, -2.196028232574463, 0.3971865773200989, -0.37300360202789307, -0.7683387994766235, -0.8716123104095459, -0.8037852048873901, -0.4596159756183624, -1.090450644493103, -1.3833149671554565, 1.3119813203811646, -1.2024272680282593, -0.8000513911247253, 0.33026811480522156, 0.3204744756221771, -0.7327943444252014, -0.061335042119026184, -0.389847993850708, -0.19536378979682922, -1.1317718029022217, 0.012283407151699066, -0.03631510213017464, 0.44349250197410583, -0.9797031283378601, 1.3750947713851929, -0.9097702503204346, 0.34089386463165283, 1.39234459400177, 1.048782467842102, 0.2279817909002304, 0.2456882894039154, -2.2590038776397705, -1.1744725704193115, -0.18783442676067352, 0.0649510845541954, -1.6131706237792969, 0.5669849514961243, 0.3009733557701111, -0.7465553283691406, -0.002491980092599988, 1.6023380756378174, 0.7768087983131409, 0.26757410168647766, -0.2387678623199463, -1.0046204328536987, 0.5487477779388428, -0.9856849312782288, 0.6328288316726685, -0.5521641969680786, 0.3243800103664398, -1.923380732536316, -0.11391914635896683, 1.1331450939178467, 0.6962422728538513, -0.4450700879096985, -0.6058819890022278, -0.2927403151988983, 0.21525435149669647, -0.28059935569763184, 0.0555424727499485, -0.5039296746253967, -0.9700629115104675, 1.862847089767456, -0.3838419020175934, 0.1318175047636032, 1.6385109424591064, -0.41690364480018616, -0.1160067468881607, 0.12684538960456848, -0.16740624606609344, -1.2282480001449585, 1.3036311864852905, -0.4153870940208435, 0.12773218750953674, 1.140924096107483, 0.9583826661109924, 0.7310709953308105, -1.8723978996276855, 0.6360892653465271, 1.7300028800964355, 0.06334398686885834, 0.3756718635559082, 0.26986080408096313, -1.2857906818389893, -1.5848350524902344, -1.5677400827407837, 0.18076974153518677, -1.2570289373397827, -0.5077123045921326, -0.45393043756484985, 1.4886356592178345, 0.2504279613494873, -0.5635062456130981, -0.5356256365776062, 0.8227329850196838, -1.0381076335906982, -1.9202725887298584, -0.8277461528778076, -0.36670228838920593, 0.7987194061279297, -0.19023677706718445, 0.049791187047958374, -0.6664398312568665, -0.5364246368408203, 1.7657849788665771, -1.7731094360351562, 0.17909689247608185, 0.05461334437131882, 1.9751832485198975, -0.7206180095672607, -1.4629487991333008, -0.06974829733371735, 1.1827986240386963, 0.6684170365333557, -1.3567249774932861, -0.5432090759277344, -0.5047101974487305, 0.6119428873062134, 1.7151390314102173, 0.10730329155921936, -1.2718546390533447, 1.967639446258545, -0.24686510860919952, -1.3583379983901978, -1.6894559860229492, -0.8294427394866943, 0.9319143295288086, -0.33212366700172424, 1.7250367403030396, -0.3869505226612091, 0.035672783851623535, -0.9791008830070496, -0.057509537786245346, -0.3743232488632202, -0.09048008173704147, -0.43568816781044006, 1.0429482460021973, 1.180113673210144, -0.8645972013473511, -1.7146531343460083, -0.03177853301167488, -0.6539179086685181, -0.043362975120544434, 1.3324134349822998, -0.27928659319877625, 0.40903493762016296, 0.7058232426643372, 1.6390098333358765, 0.694923996925354, -0.835621178150177, -0.9758045077323914, -0.9277464151382446, 0.12237493693828583, -0.4780460298061371, -0.4573068618774414, -0.2597960829734802, 0.07280635088682175, -0.4529205560684204, -1.3706021308898926, 0.19459202885627747, 1.1427814960479736, 1.3046976327896118, 0.6725142598152161, 2.35309100151062, -2.1556057929992676, -0.8282800316810608, 1.0292733907699585, 0.582994818687439, 0.8682205080986023, 0.2933858036994934, 2.5146756172180176, 1.5594254732131958, 0.3005627393722534, -2.1989476680755615, -1.4930247068405151, 0.9699452519416809, 2.6007065773010254, -0.26252055168151855, 0.5708662867546082, -0.5715268850326538, -0.6590766310691833, 0.5337139368057251, 0.04865182936191559, -1.3908331394195557, -2.3032166957855225, -0.09991122037172318, 2.695582151412964, 0.5498801469802856, -0.4547707736492157, 2.158700942993164, 1.2024263143539429, 0.6810740232467651, 0.12336111068725586, -0.45622697472572327, -0.7077673673629761, 0.053397975862026215, -0.2867633104324341, 0.23245878517627716, -0.8642985224723816, -1.1635445356369019, 0.32931700348854065, 0.30744311213493347, 1.2504942417144775, -0.29215237498283386, -0.7655156254768372, 0.7189173102378845, -0.5497138500213623, 0.37218254804611206, 1.2724952697753906, -0.609931468963623, -0.4274041950702667, -0.3926132917404175, -0.38852202892303467, 0.23788435757160187, 0.46556851267814636, -0.1133013516664505, 0.3056119680404663, -0.20467548072338104, -1.6567896604537964, -0.36798539757728577, 0.8738109469413757, -0.82525634765625, -0.7396657466888428, -0.10764134675264359, 0.43017664551734924, 0.2743338644504547, 0.7211515307426453, 1.6261957883834839, 0.38746270537376404, -0.3518437445163727, 0.07379867136478424, 1.349707841873169, -0.0710444524884224, -0.5541325211524963, 0.8204536437988281, 0.04423021152615547, 0.795319139957428, 0.5538880825042725, -0.673165500164032, -0.3865363895893097, 0.8172680139541626, -0.7137993574142456, -1.1337039470672607, -2.6651244163513184, -0.003233895869925618, -0.4696328043937683, 0.021674644201993942, 0.21644824743270874, -1.6633754968643188, 0.8750612735748291, -1.3732020854949951, 0.5199490189552307, -0.008670195937156677, -0.22777876257896423, -0.9281423091888428, 1.322535753250122, 0.2418140172958374, -0.8446672558784485, 0.5795811414718628, 0.33164921402931213, -0.3513638973236084, 0.20145076513290405, -0.4927504062652588, -0.6115119457244873, 0.4815142750740051, 0.12150941789150238, -0.276980459690094, -0.9259645342826843, -0.7029752135276794, 0.5510914325714111, -1.9024204015731812, -0.6090146899223328, -1.1866083145141602, 0.026710262522101402, -1.816004753112793, 1.1564189195632935, -0.5725165009498596, 0.7176262140274048, 0.7844119071960449, 0.02575679123401642, 0.28660768270492554, 1.055202841758728, 0.6601807475090027, -0.2260466068983078, -0.23279684782028198, -0.8482441902160645, -0.5479705333709717, 0.6802120804786682, 0.026724200695753098, -0.4401603043079376, -1.7981685400009155, 1.326711893081665, 0.22546736896038055, -1.2462366819381714, -1.860743761062622, -0.23214448988437653, 0.696563720703125, 1.0205683708190918, -1.0135334730148315, 0.8703060746192932, -1.6259225606918335, 0.571962833404541, 0.01653221622109413, 0.14585749804973602, -1.6793328523635864, -1.3070038557052612, -0.4888797998428345, -0.11060134321451187, -0.4330834448337555, 0.49556025862693787, 0.8257237672805786, 0.5497260689735413, 2.5405261516571045, -0.5166633129119873, -1.5560866594314575, 0.3592261075973511, -0.2928220331668854, 1.39848792552948, -0.3680651783943176, 0.15352164208889008, 0.12206888198852539, -0.37847137451171875, 0.3041445314884186, 2.213090181350708, 1.1990886926651, -2.296966075897217, 0.32287105917930603, -0.654003918170929, -2.2582786083221436, 1.3663859367370605, -0.7717748880386353, -0.19030795991420746, -2.2298591136932373, -0.9114015698432922, -2.6030304431915283, -0.12695883214473724, -0.503267765045166, 1.5058104991912842, 0.7705478072166443, 0.14681129157543182, -1.375780463218689, -0.21662092208862305, 0.15307652950286865, -0.31248095631599426, 0.7410584092140198, -0.7133135795593262, 0.4460984766483307, 0.26192137598991394, 2.403454542160034, 0.24206708371639252, 0.5525369644165039, -1.5687036514282227, -0.860759973526001, 0.2371925711631775, -0.6075043082237244, -0.7706941962242126, 0.31765875220298767, -1.3739761114120483, -0.27644333243370056, 0.8283340930938721, -0.20862925052642822, -1.393112301826477, 0.6680074334144592, -0.48655223846435547, -0.4104604423046112, -0.7240877151489258, 0.1458958238363266, 0.07811151444911957, 0.2143816202878952, 1.089451789855957, 0.7530127763748169, -0.25136637687683105, 0.09163907170295715, 0.007822425104677677, -0.7068598866462708, -0.5947778224945068, 0.4585782587528229, -1.2433371543884277, 1.1359174251556396, 2.211536407470703, -1.1952104568481445, -0.3296422064304352, -0.3446413576602936, 0.15091544389724731, -1.165796160697937, -0.9008765816688538, 0.01183281559497118, 0.10172653198242188, -0.8700960278511047, 1.2478317022323608, 0.6682383418083191, 1.482741117477417, -1.8519155979156494, -1.31220281124115, 2.131572961807251, 0.01451409887522459, 1.3408178091049194, 0.9489455223083496, -1.0570260286331177, 0.32558006048202515, 0.5833123326301575, -0.2762744426727295, 0.18108689785003662, -0.15133371949195862, 0.6336098313331604, 1.3852357864379883, -1.2784565687179565, -0.035929590463638306, -1.5352154970169067, -0.9502794742584229, 0.12038524448871613, -1.7498286962509155, 0.3398236930370331, -1.3289082050323486, 1.3613933324813843, 0.17883992195129395, 0.3533133864402771, 0.5482674241065979, 0.45532238483428955, 0.5569348931312561, 0.4313645660877228, -1.661497950553894, 0.5703537464141846, 1.4599875211715698, -1.1449909210205078, 0.059146199375391006, -1.464032530784607, -0.6039016842842102, -0.5855419635772705, -0.19576045870780945, -0.6446054577827454, 0.8044257760047913, -0.7221008539199829, -0.7854094505310059, -0.3107149004936218, 0.8713087439537048, 1.1314771175384521, -1.1885972023010254, 0.5135715007781982, -0.004341356921941042, 0.7796701192855835, -0.8082799911499023, 0.2501426041126251, -0.09435033798217773, -0.21813353896141052, 0.8135313987731934, 0.8751962184906006, -0.44150617718696594, -0.2727113664150238, 0.4555899202823639, -0.6950814127922058, -1.7124807834625244, -0.5972169637680054, -0.31385523080825806, -0.46673908829689026, -1.4323394298553467, 0.21838459372520447, -0.24818816781044006, -0.321413516998291, -1.0392099618911743, 1.4162225723266602, 1.6163866519927979, 0.2531164586544037, 0.2843945622444153, -0.2614651322364807, -0.20150108635425568, 0.6637258529663086, -0.5255917906761169, 0.5347378253936768, 0.9847793579101562, -0.44270825386047363, 1.8192493915557861, 0.012321419082581997, 1.8970012664794922, 1.0813608169555664, -0.717829167842865, 1.3697104454040527, 0.4391483664512634, 1.5721869468688965, 0.2637823522090912, 1.3601244688034058, -2.053112506866455, -0.4694366753101349, -0.7208461761474609, 0.446736603975296, -0.9722652435302734, -0.701199471950531, 0.1690538227558136, 0.5807533860206604, -1.5033823251724243, 1.1049575805664062, -1.6451363563537598, 0.11940055340528488, 0.8641331195831299, -0.9496492743492126, -0.05197814106941223, -0.6205008625984192, -0.24500158429145813, -0.29300060868263245, 0.24608643352985382, -0.07328873872756958, 0.664871335029602, 2.014146566390991, 0.2433912605047226, 0.6100186109542847, -0.05769560858607292, -0.3081616461277008, -2.425921678543091, -0.5888347625732422, 0.4829210340976715, -0.612001359462738, -0.4441262483596802, -0.06546548008918762, 0.10952214151620865, -0.27005571126937866, -2.1792521476745605, -0.8199043273925781, 0.25827687978744507, 0.09685776382684708, -1.097861886024475, 0.8213793635368347, 0.383835107088089, -0.6031755208969116, 0.9839392304420471, 1.0998806953430176, -0.6492400169372559, 0.06966925412416458, 1.2252455949783325, 0.5763372182846069, 1.0701863765716553, -2.263230323791504, -0.8659129738807678, 1.002902626991272, 0.3939085900783539, 0.16118007898330688, -0.47280165553092957, -0.19053448736667633, -1.070014238357544, -1.8514971733093262, -0.638519823551178, 0.15005196630954742, 0.5933046340942383, -0.027317499741911888, 0.4061551094055176, -0.4093664288520813, 0.6489083766937256, 1.6028417348861694, 1.144650936126709, 1.4190832376480103, 0.4884183704853058, -1.8495219945907593, -0.5705885887145996, 1.1855857372283936, 1.2257580757141113, 0.43365344405174255, 1.242047905921936, -1.7214471101760864, 1.656795620918274, 1.096274733543396, -1.8991122245788574, -1.188718318939209, -1.2694594860076904, 1.239784836769104, -0.8218487501144409, 0.40442338585853577, -1.857154130935669, 0.6743517518043518, 0.9086383581161499, 0.2987503707408905, 0.2941771447658539, -0.9301294684410095, 0.3647424578666687, -0.15615113079547882, -0.3873511552810669, -0.27644601464271545, 0.9872705340385437, 0.4157944917678833, -0.6119146943092346, 1.2858963012695312, -2.666611433029175, 0.12491359561681747, -0.9701249003410339, 2.1555511951446533, 0.4208679795265198, -0.04471733048558235, -0.6895930767059326, 1.7140685319900513, 1.3407305479049683, 1.0838831663131714, -0.358854204416275, 0.2493826299905777, -0.14933554828166962, -1.0593215227127075, -1.1302310228347778, 0.08412609994411469, 1.2376294136047363, 0.06605791300535202, 0.6013851761817932, -0.7963281273841858, 1.0126913785934448, -0.6571577191352844, 0.9794794917106628, 0.1347200721502304, -0.9845290780067444, 1.5482292175292969, -0.9075961709022522, -0.22705772519111633, -0.8160107135772705, 1.984537124633789, 0.22974561154842377, 0.5152220726013184, 0.5554154515266418, -0.25482177734375, -0.2577746510505676, 1.8038527965545654, -0.6571414470672607, 1.943852186203003, -1.4191181659698486, 0.18601737916469574, 0.09857062995433807, 0.5164961218833923, 0.005230993963778019, -0.30861708521842957, 0.4511553943157196, -0.4708099067211151, -1.2236227989196777, -0.3724425137042999, 0.4073069989681244, 0.09221185743808746, 0.028701310977339745, 0.6855895519256592, -1.0350102186203003, 0.10635640472173691, -0.24359038472175598, 0.18063890933990479, -1.0028265714645386, 0.6340732574462891, 0.5632616877555847, -0.10929326713085175, -1.6728497743606567, 2.3124358654022217, -1.6396188735961914, 0.5276318192481995, 0.1560724377632141, -0.7851582765579224, -0.1338617503643036, -0.9111388921737671, -0.12346149235963821, 0.45318785309791565, -0.2467772364616394, -0.5914375185966492, -0.24383622407913208, 1.225616455078125, -1.7670953273773193, -1.2384463548660278, 0.5373529195785522, -1.709060788154602, -0.2842170000076294, -0.39191296696662903, 1.288262128829956, -0.3963039219379425, -0.24273940920829773, 0.48916709423065186, 0.47427958250045776, -1.8860946893692017, 0.6177245378494263, -0.3564474582672119, 1.5348634719848633, -0.4072045683860779, 0.5016247034072876, 1.0595383644104004, -0.9121551513671875, 0.8248981237411499, -0.5602863430976868, 1.7794026136398315, -0.5409727096557617, 1.401676058769226, 0.7776908874511719, -0.6104809641838074, -0.7481024861335754, -0.23908619582653046, 1.2423378229141235, -0.5408385396003723, 0.13801681995391846, 0.30806705355644226, -0.398867666721344, 0.4598538875579834, 2.2640841007232666, -0.09449264407157898, 1.5768760442733765, 0.9661956429481506, 1.4123011827468872, -0.14667393267154694, 0.1028447300195694, -1.168639898300171, 0.6994081735610962, -0.02185859903693199, 0.7058160305023193, 0.26957234740257263, 1.2521101236343384, 0.6742805242538452, -0.01205478049814701, 1.1229208707809448, -1.3114988803863525, 1.3999632596969604, 0.04647966846823692, -1.6181493997573853, -0.26422014832496643, 0.9393379092216492, -0.5858285427093506, -0.15219397842884064, 0.8963497877120972, 2.7382748126983643, 0.6200839281082153, 1.4543639421463013, 1.7932307720184326, -2.184561014175415, -1.122516393661499, 0.3583129942417145, -1.816541075706482, -1.7216211557388306, 1.1472262144088745, -0.8434416055679321, 0.5053666830062866, 0.30184268951416016, -0.1440633237361908, 0.08534736931324005, -0.32961103320121765, -1.7379717826843262, -1.4948772192001343, 1.367210030555725, -0.3495210111141205, 0.6342843174934387, 0.5017841458320618, 0.6347370743751526, 0.7370640635490417, 1.9844655990600586, 1.0536527633666992, -0.10385502874851227, 1.504118800163269, 0.11006228625774384, -0.39423346519470215, -0.3562738597393036, -0.07928466796875, 1.7142082452774048, 0.6601759195327759, 1.5602455139160156, 1.7864456176757812, -1.493669033050537, 0.9959092140197754, -0.2363533079624176, -1.1132404804229736, -0.457518607378006, 1.1100438833236694, -0.14297929406166077, -0.37726232409477234, -0.9778750538825989, -0.615874707698822, 1.0274566411972046, -1.447572946548462, -0.17845362424850464, -0.3767155706882477, 0.29656124114990234, -1.2730780839920044, -0.6564204692840576, -0.6922609210014343, 0.13846617937088013, 0.8969020843505859, 0.46942585706710815, -2.0327999591827393, -0.018719367682933807, 0.7876980304718018, -0.644709050655365, -0.7377091646194458, -1.2043826580047607, 0.4623467028141022, -0.4471225142478943, -2.5392017364501953, -0.927717924118042, 0.11550969630479813, 1.1487579345703125, -0.33626726269721985, -1.057183027267456, 1.4013835191726685, -0.8649556636810303, 1.4030933380126953, 0.07405611872673035, 0.012201393023133278, -1.0640273094177246, 0.09015006572008133, 1.0839310884475708, -0.7914291620254517, 0.0855783149600029, -0.1585293710231781, -0.39122793078422546, -0.9290196895599365, -0.12175916135311127, -0.44024381041526794, 0.632697582244873, -0.6415899991989136, -0.1238207295536995, 0.44427093863487244, -0.06879188865423203, -1.2082465887069702, -1.3315246105194092, -1.0766488313674927, 0.3572612702846527, -1.033995509147644, -1.286500334739685, 2.2067959308624268, 0.648210883140564, 1.1748688220977783, -2.2225892543792725, -0.941738486289978, -0.17129208147525787, -0.7797005772590637, 0.5570324659347534, 0.4438953995704651, 0.511196494102478, 1.1147500276565552, -0.4990312159061432, 0.44135361909866333, 1.2353874444961548, 0.20446375012397766, 0.2758635878562927, -0.2591284215450287, 0.7192010879516602, -0.25634676218032837, 0.6305810213088989, -0.3282099962234497, -0.7566540837287903, 1.868057370185852, 0.10151661187410355, -0.4134399890899658, -1.3407721519470215, -0.697027325630188, 2.041234016418457, 1.3473100662231445, 1.1300687789916992, 0.3186348080635071, 0.046016525477170944, 1.78928542137146, -0.9609184265136719, 0.2068668007850647, 0.7617405652999878, -0.868514895439148, -0.881479024887085, -0.20038607716560364, 0.4452311396598816, -0.3638729155063629, 0.42267876863479614, -1.738065242767334, 0.9769734144210815, -0.251723974943161, -0.6148957014083862, 1.9968622922897339, -0.26895594596862793, -0.6247469782829285, 0.7602323889732361, -0.3150089979171753, -1.2612926959991455, -0.5107945799827576, -0.486835241317749, 0.42769062519073486, -0.06436319649219513, -2.0037264823913574, -0.07100280374288559, 0.7325626015663147, 0.007764932233840227, -1.5061614513397217, -0.6564481854438782, -0.06689178198575974, -0.5339502096176147, 1.382571816444397, 0.07594973593950272, -0.3019469976425171, -1.1611307859420776, -0.1994621306657791, -0.5169425010681152, 0.8596121072769165, 0.7228826284408569, -0.9619860649108887, 1.088213324546814, -0.2620307207107544, 0.9070602655410767, 0.3415333926677704, -0.08861635625362396, -0.08115491271018982, -0.7550137042999268, -0.29513460397720337, 0.7997401356697083, -0.8332300186157227, 0.08273205906152725, -1.1684950590133667, -1.0804784297943115, -0.801412045955658, -1.9714882373809814, 0.8000898957252502, 0.28012725710868835, -0.22898772358894348, 0.2893573045730591, 1.283436894416809, -0.549176812171936, -1.5489773750305176, -0.4305919408798218, -0.1829327642917633, 0.08451325446367264, -0.10411342978477478, -0.2427232414484024, -2.526235342025757, -1.0777827501296997, -0.16622895002365112, -0.8069680333137512, 2.4233691692352295, 1.509555697441101, 0.5462006330490112, 0.11143769323825836, 1.1379348039627075, 0.09289168566465378, 0.5009345412254333, -0.18701960146427155, 0.32594648003578186, -0.4486846625804901, -0.3160456120967865, 2.1313295364379883, -0.07712067663669586, -0.10350623726844788, -0.6780387163162231, 1.4786481857299805, -1.1446706056594849, 0.7189924120903015, -0.23688696324825287, 0.18776367604732513, -0.682259202003479, 0.019223906099796295, 0.47891172766685486, -0.06248132511973381, -0.25440409779548645, -0.8343110680580139, 0.3252202868461609, -1.0024306774139404, 0.3297281265258789, 1.1886972188949585, 0.6286271810531616, -0.8374176025390625, 0.06506141275167465, -0.6164808869361877, 0.5088199377059937, 0.5688467621803284, -1.5497561693191528, 0.06845857948064804, -0.6094719767570496, -0.803895890712738, -0.8307778239250183, 0.4598104953765869, -0.09849872440099716, 0.23487064242362976, 0.5577390193939209, -0.19077637791633606, -1.7243239879608154, 0.5781745314598083, -0.7869846820831299, 0.5925027132034302, -0.652326762676239, -0.23576317727565765, -1.602586269378662, 1.7766703367233276, 0.24615347385406494, -0.8255402445793152, -0.40994054079055786, -0.5733014941215515, 0.650641679763794, -1.023862600326538, -0.39682939648628235, 0.7573744058609009, -0.9525073766708374, 0.8775128126144409, 0.007803576532751322, 2.1796395778656006, 0.3799404501914978, 0.6385781168937683, -0.4300474524497986, -0.3254513442516327, 0.10109730064868927, -0.9377577900886536, -1.3442912101745605, -0.9727285504341125, -0.03523065894842148, -0.4080425798892975, -1.1093560457229614, 0.24202801287174225, 0.086395263671875, 0.5097022652626038, -1.1828620433807373, -0.15420566499233246, -0.4284772276878357, -0.7494879364967346, 0.06119317561388016, 1.207301139831543, 0.04167570173740387, 0.25557711720466614, 1.6578606367111206, -0.9161748886108398, 0.753283679485321, -0.8733989000320435, 0.8338748216629028, -1.5440611839294434, 0.9338422417640686, -0.009927061386406422, 0.5808720588684082, -0.6196430325508118, -1.212670922279358, -2.7440335750579834, -0.2696581184864044, -1.2987159490585327, -0.7105566263198853, 0.3319154381752014, -1.1691083908081055, -0.06392774730920792, -0.3963375687599182, 0.5933650135993958, 0.712398886680603, 1.3748937845230103, -0.8614526391029358, -0.1687282919883728, 1.4712443351745605, 0.9426857829093933, 0.9170145988464355, 0.44260457158088684, 1.3981353044509888, -0.5054318308830261, 0.17779441177845, 1.6404924392700195, -1.789623498916626, -0.7939754724502563, 0.9445487260818481, -0.10405615717172623, -1.373299479484558, -1.5643858909606934, 0.6156871318817139, -0.790910005569458, -0.5232803225517273, 0.10019568353891373, 0.23332911729812622, -0.6084703803062439, -3.03937029838562, -0.13022646307945251, 1.1580355167388916, -1.177223801612854, -0.5997672080993652, 0.7093627452850342, 0.11529767513275146, 0.16058871150016785, 0.0704808309674263, 1.6943886280059814, -0.5510290265083313, -0.3824273645877838, 1.671354055404663, 1.7140617370605469, 0.9582141637802124, 1.0633705854415894, -1.1033744812011719, -0.0035462449304759502, -0.8480597734451294, 1.4350183010101318, 0.16209417581558228, -1.0223093032836914, -2.1030189990997314, -0.6139468550682068, 1.4897438287734985, -0.38117659091949463, -0.3680841624736786, -1.5650957822799683, -0.5410099625587463, -2.511545419692993, 0.06830286979675293, -0.04344576597213745, -0.14686977863311768, 0.6041630506515503, 0.7516777515411377, -2.094383478164673, -0.5828114748001099, -1.5753198862075806, -1.7925769090652466, 0.6777064204216003, 0.4354613125324249, -0.6686331033706665, -1.3010902404785156, -0.19172731041908264, -0.10498958826065063, -0.6721416115760803, 1.844642162322998, 0.04122856259346008, -1.4551987648010254, 1.9573665857315063, -0.6116671562194824, -1.3370798826217651, 0.35102444887161255, -0.649064838886261, 0.2848062813282013, -1.1156030893325806, 0.9779579639434814, -0.786435604095459, -1.4761360883712769, 1.6215510368347168, -1.1975663900375366, 2.2072761058807373, 1.1979116201400757, -0.39029479026794434, -1.1214613914489746, 0.294678270816803, -0.662116527557373, -0.7556828260421753, 0.07738342881202698, -1.4781638383865356, -1.0602797269821167, 0.1850384622812271, 0.8055990934371948, -0.35643818974494934, 0.02315473183989525, 0.43332815170288086, 0.7787806391716003, -0.8732393980026245, 1.083591103553772, -1.223305106163025, -1.8894511461257935, 0.03373537212610245, -1.4848843812942505, -0.9010837078094482, 0.224992036819458, 0.8390522003173828, -0.628740668296814, -0.020687628537416458, 0.3285461962223053, 0.1794222593307495, -0.2822677791118622, -0.4997372329235077]], "dtype": "float32"}, {"__nd__": [[-0.062099505215883255, -0.3485293388366699, -1.0565824508666992, -0.817670464515686, -1.4958984851837158, -0.7912722826004028, -0.25940388441085815, -0.8585463762283325, -0.3977336585521698, 1.1242389678955078, 1.7073307037353516, -3.304611921310425, 0.23496751487255096, 0.1586204618215561, -0.298041969537735, 1.3757034540176392, 1.3022958040237427, -0.7652437090873718, -0.12912224233150482, 0.025123493745923042, -2.3827145099639893, -0.23524531722068787, 0.20182478427886963, -0.6927984356880188, 0.5714287161827087, 0.12622523307800293, 1.6704896688461304, -0.15296848118305206, -0.5967950224876404, -0.040466997772455215, -0.016686219722032547, 0.859632670879364, 0.7195544242858887, -1.0461441278457642, 1.4993245601654053, 0.8680142164230347, -2.1016364097595215, 0.3988790214061737, 0.7553678154945374, -0.125550776720047, -0.8616957068443298, -1.4099535942077637, 0.4915219843387604, 1.003427267074585, -0.08419289439916611, 0.8106017708778381, 2.3258275985717773, -0.30670779943466187, 1.1070483922958374, 0.6121790409088135, 0.7986230254173279, -0.969998836517334, -0.4657464921474457, 0.5306046605110168, 0.20011815428733826, -0.3778291344642639, -0.14857041835784912, 1.7074917554855347, 1.3255903720855713, -0.15020576119422913, 0.6774365305900574, -0.6627615094184875, 0.3866415321826935, -0.8009944558143616], [-0.1296471655368805, 0.7891255021095276, -0.05773252993822098, 0.23751875758171082, 0.8812693357467651, -0.4293863773345947, 0.447228342294693, -0.5759842395782471, 0.08767643570899963, 1.0916261672973633, 0.006911137141287327, -0.9596002697944641, 0.8383910655975342, 0.8853504061698914, -0.05432770773768425, -0.42054983973503113, -0.2860482633113861, -0.26233941316604614, -0.11453194916248322, -0.9144983291625977, 0.13078302145004272, -0.05344577133655548, 0.2166755646467209, 1.1335710287094116, -0.25594690442085266, 1.1438461542129517, -1.0548934936523438, -1.1144187450408936, -0.13729611039161682, 0.4415951669216156, 0.242884561419487, 1.1313952207565308, 0.2855933606624603, 1.0743497610092163, 1.6164499521255493, 0.13962027430534363, 0.2028665542602539, -1.1153093576431274, -1.2093111276626587, 0.7601767778396606, 1.5765331983566284, 1.15253484249115, -0.11166076362133026, 0.4876256287097931, 0.5777403712272644, 1.3177847862243652, 0.8471344709396362, -1.3437689542770386, 1.3057971000671387, -1.07182776927948, 0.010838806629180908, -0.6100035905838013, -0.9131996631622314, 0.5476524829864502, -1.876858115196228, -0.5935626029968262, -0.2894549071788788, 0.1425347626209259, 0.787381112575531, -0.13690602779388428, 0.5856468081474304, -1.2455817461013794, -0.5475735664367676, 0.7285938262939453], [0.24032458662986755, 1.3780304193496704, 1.0500274896621704, -0.08104115724563599, -1.5521931648254395, 0.18694578111171722, -1.6477434635162354, 0.6844277381896973, 0.5303261280059814, 0.5280281901359558, -0.5719919800758362, -0.5035942196846008, 1.1946654319763184, 0.8166055679321289, 0.9929420948028564, 0.85368412733078, 1.800888180732727, -0.21655920147895813, 0.0074202390387654305, -0.1464919000864029, 0.3251539468765259, 0.05836927518248558, -0.17942044138908386, 0.5822234749794006, -0.42735201120376587, -0.018735231831669807, -0.20796677470207214, -0.4520839750766754, 0.6250515580177307, -0.1693762242794037, 1.2609188556671143, -0.22296679019927979, 0.7775450348854065, -2.652446985244751, 0.05493689700961113, 0.02082843706011772, -0.3681940734386444, 1.2140141725540161, 0.3918164372444153, 0.8552446961402893, 0.10023584961891174, -0.9031639099121094, 0.16254645586013794, 0.24116568267345428, 1.2448850870132446, 1.580810308456421, 0.8008068799972534, -0.29412442445755005, -2.8978028297424316, 1.5597468614578247, 1.6690635681152344, -1.3172858953475952, -1.194072961807251, -0.4837619364261627, -0.2788299024105072, 0.2354227751493454, -1.3521207571029663, 0.05422060191631317, 0.9991676807403564, -0.48255056142807007, 0.7483518719673157, -0.37871676683425903, -0.3815952241420746, 2.2423763275146484], [-2.0330705642700195, 1.0258492231369019, -1.9900166988372803, -1.1747467517852783, -1.0268360376358032, -0.8178717494010925, -0.013404789380729198, 1.3882094621658325, 0.16136230528354645, 1.6687986850738525, 0.14578858017921448, -0.6577662229537964, 0.8315430879592896, 1.1139277219772339, 1.0946297645568848, -0.11439725011587143, 0.5442690849304199, 1.1833736896514893, -1.0069607496261597, 0.682077944278717, 1.194455862045288, 0.07052261382341385, -1.1889971494674683, 0.24926987290382385, 1.1715173721313477, -0.7873862385749817, -0.9640724658966064, -1.4749327898025513, 0.1659248322248459, 0.08073403686285019, 2.009547472000122, -0.04673450067639351, 0.9716487526893616, -0.5328274369239807, -0.39677655696868896, 0.8734161853790283, 0.4837075471878052, 1.6578489542007446, 0.027783088386058807, -0.7987997531890869, -1.1452239751815796, -0.4352932870388031, -0.5137410163879395, -1.0916639566421509, 1.3628438711166382, 2.4219980239868164, -1.0942317247390747, -0.29963788390159607, -3.0368454456329346, 0.7767128348350525, 0.5941054224967957, 1.086584448814392, -0.14567795395851135, 1.0473815202713013, -0.21408741176128387, -1.4899522066116333, 1.387555480003357, 1.1072981357574463, -0.9506130218505859, -0.580752432346344, 1.5313503742218018, -1.5059388875961304, -1.0850212574005127, 0.20517712831497192], [0.32346662878990173, 1.3586249351501465, -0.017671866342425346, -0.29042017459869385, -0.19030669331550598, 1.0037130117416382, 0.7520459890365601, -0.3674676716327667, -1.9903459548950195, -0.5472840070724487, 0.29667967557907104, -1.6267890930175781, 1.8815642595291138, -0.9580225944519043, 0.5544291138648987, 0.2533336579799652, -0.6260574460029602, 1.8118525743484497, -0.3007516860961914, 1.0760257244110107, -0.0028547169640660286, -0.7903456091880798, -0.045587316155433655, -0.5103586912155151, -0.6366819143295288, 0.6161254644393921, 0.22433319687843323, 0.8082976341247559, -0.10782549530267715, -1.5598303079605103, 0.1901426762342453, 0.9363325238227844, -0.24609479308128357, 1.0874488353729248, -0.27742084860801697, 0.2881282567977905, -0.20211391150951385, 0.03644460812211037, 0.10881514847278595, 0.32897669076919556, -0.11958777904510498, -0.19651484489440918, -0.6107505559921265, -1.4791148900985718, -1.1853585243225098, -0.03183431178331375, -3.0922133922576904, 0.16029353439807892, 1.7564449310302734, 1.367680549621582, 0.7989068627357483, 0.5885043144226074, -0.7801279425621033, 0.20241084694862366, -0.8090984225273132, -0.2703753113746643, 0.4507364332675934, 1.249263048171997, 0.25678014755249023, 1.1329478025436401, -0.5266329050064087, -1.690527081489563, 0.3002234995365143, -1.0197499990463257], [1.0335503816604614, -1.3892724514007568, -0.11356427520513535, 1.1847054958343506, 0.4954044222831726, 0.7551396489143372, 1.8295146226882935, 1.0670853853225708, 1.8802165985107422, -0.4970977306365967, 0.8152899146080017, -0.012947344221174717, -0.40191027522087097, 1.5607026815414429, -1.607138752937317, 1.1437052488327026, 1.227768898010254, -0.12044310569763184, -0.053349003195762634, -0.039111435413360596, 0.47428345680236816, -1.4448896646499634, 0.37446630001068115, 0.32546278834342957, -1.0965361595153809, -0.04722519963979721, -0.35075294971466064, -0.6773529648780823, -0.6992661356925964, -0.3968818783760071, 0.7192308902740479, -0.9191123843193054, 0.16229946911334991, 1.4098258018493652, -1.1495169401168823, 0.11773542314767838, 0.904339075088501, -0.9903467893600464, -0.45413821935653687, -0.8236269950866699, -1.0489221811294556, -1.4128646850585938, -0.267721563577652, -0.3051719069480896, -2.2854301929473877, 0.2689667344093323, 1.4983177185058594, 1.2064757347106934, -1.7258816957473755, -0.09809702634811401, -1.768345832824707, 0.38678547739982605, -1.6468600034713745, 0.23979580402374268, -1.4119150638580322, -1.948810338973999, -3.8036372661590576, -0.4448651075363159, -0.31151312589645386, 0.7609959244728088, -0.9746397733688354, 0.6194165349006653, -0.30490627884864807, -0.9742046594619751], [-0.24653151631355286, -1.1371663808822632, 1.1636046171188354, -0.45242080092430115, 1.2050015926361084, 1.5763232707977295, -0.08195064961910248, 0.5879331231117249, -0.7826419472694397, 0.569663941860199, 1.5601482391357422, 0.5668231844902039, -0.5478444695472717, 0.8291798233985901, 1.0666738748550415, 0.15900981426239014, -0.032434217631816864, -2.250420570373535, 0.0694284439086914, 0.9525641798973083, 0.6841475963592529, -0.08675810694694519, -0.43421733379364014, -1.7458572387695312, 1.3608372211456299, 0.5461612939834595, 0.11199493706226349, -0.35560035705566406, 0.001746053691022098, 1.3426775932312012, 0.18636466562747955, -0.3619861304759979, -1.215864658355713, -0.3072701096534729, 1.4652711153030396, -0.9095838665962219, -0.5960531830787659, 1.3799552917480469, -0.5634946823120117, 0.2848470211029053, 0.39660295844078064, -0.6371439695358276, 0.6588519811630249, -0.8301988840103149, 0.9353890419006348, -1.0663113594055176, 0.5218891501426697, 0.24473293125629425, -1.0654168128967285, -0.2654995918273926, -1.0217885971069336, -0.3660188317298889, -0.9288519620895386, -0.1566539853811264, 0.7642930746078491, 0.504745602607727, -0.31127074360847473, 1.44256591796875, 0.9148194193840027, 0.1504981964826584, -0.9491746425628662, 1.2968157529830933, 0.29502642154693604, -0.3901119530200958], [0.8807081580162048, -0.9304506182670593, -0.6096529364585876, 1.2517163753509521, 1.0137256383895874, -1.8082841634750366, 1.045952558517456, 0.18429695069789886, -2.3199291229248047, -0.15231850743293762, -0.7105326652526855, -0.17145627737045288, -2.8692307472229004, -0.2858402729034424, 0.5929098725318909, -0.5617685914039612, -0.1734299510717392, -0.3210737407207489, 1.2693966627120972, 1.233282208442688, -0.8428082466125488, -0.44286632537841797, -1.6112840175628662, 1.037095546722412, -1.479821801185608, 0.3323470652103424, -0.5051582455635071, -0.013426787219941616, 1.3456963300704956, 0.8692524433135986, -0.45674148201942444, -0.23842668533325195, 0.23033863306045532, -0.5939577221870422, -1.6973897218704224, -0.3350332975387573, 0.9637287259101868, 1.636355996131897, -0.8338424563407898, 0.04468771070241928, -0.24103209376335144, -0.02227921597659588, -0.36233314871788025, 0.47540658712387085, 0.03821619227528572, 1.2854284048080444, 0.06835390627384186, 0.02651558257639408, 1.1059209108352661, -1.8874495029449463, 0.2304859608411789, -0.0016081674257293344, 1.1023112535476685, 0.09620629251003265, -0.5610491037368774, 0.8763178586959839, 0.31687358021736145, -0.7039956450462341, 0.131686270236969, 1.977380394935608, -1.952335238456726, 1.5128884315490723, -1.0345269441604614, -0.8192857503890991], [-0.191557839512825, 1.4272595643997192, 0.40883752703666687, 0.6581935286521912, -0.1215500757098198, 0.794851541519165, 0.9320152997970581, -0.4770258665084839, 0.5889642238616943, 0.2977883815765381, 1.239892840385437, -0.12824347615242004, 1.31884765625, 0.6624568104743958, 0.2826777994632721, -0.48060715198516846, 0.8906774520874023, 0.8407155871391296, 0.7356508374214172, 0.25773492455482483, 0.05902816355228424, -0.0876937061548233, -0.9449776411056519, 0.5754413604736328, 0.1463545709848404, 1.8454041481018066, 0.017731742933392525, -0.6256660223007202, 1.823022723197937, 0.17114536464214325, 0.040000125765800476, 0.16544625163078308, -1.34901762008667, 0.9573134183883667, -1.6449552774429321, -0.32475391030311584, 0.5319585204124451, 0.5425732135772705, 0.450715571641922, 1.3979382514953613, -1.2916083335876465, 0.3051701784133911, 0.2887914478778839, -0.3664669096469879, -0.4303332269191742, 0.08306222409009933, -0.7090960144996643, 0.6914153695106506, -1.3708416223526, -0.5704081058502197, -0.21027223765850067, -1.4756712913513184, -0.8385850191116333, 1.6101876497268677, 1.4885871410369873, 0.653510570526123, 0.15311090648174286, 1.1738837957382202, 1.0827308893203735, -0.10749895125627518, -0.3331100046634674, 0.3840698301792145, 1.8017820119857788, -0.8369589447975159], [-0.41080108284950256, -0.37583211064338684, -0.7185364365577698, 0.006328148767352104, -1.0431218147277832, 0.6275271773338318, -0.398647278547287, -0.19858890771865845, -1.6067235469818115, 1.1112899780273438, 1.4813956022262573, 1.0338252782821655, -1.004701852798462, -0.9728121161460876, 0.5676144957542419, 0.18485033512115479, 1.1475828886032104, 1.0789320468902588, 0.08008205890655518, 0.1837441325187683, -0.7792441844940186, 0.4971799850463867, 0.28307628631591797, 0.9537373185157776, -0.1500510573387146, 2.616140604019165, 0.5885253548622131, 0.8837856650352478, -0.44113966822624207, 2.4226369857788086, 0.6231056451797485, -1.037480354309082, 0.21174046397209167, -0.7692232728004456, 0.48749685287475586, -0.0600040927529335, 0.20188477635383606, 0.31098127365112305, -0.3614397644996643, -0.09958872199058533, 0.9924566745758057, -0.12668831646442413, 1.0465501546859741, 1.535682201385498, 1.0828684568405151, 0.5190234184265137, 0.27263519167900085, 0.881502091884613, -0.8103519678115845, -0.6760834455490112, -0.24384254217147827, -0.7359685301780701, 1.8869792222976685, 0.16997534036636353, -0.2343701273202896, -0.5746172070503235, -1.6989529132843018, 0.55024254322052, 0.7249997854232788, 0.34286239743232727, -0.12522517144680023, 1.7644230127334595, 1.804474949836731, 0.1761387139558792], [1.3997392654418945, 0.4687122702598572, 0.06556788086891174, -0.3884306848049164, 0.0008183076861314476, -0.9129235744476318, 1.1352332830429077, 0.8983503580093384, -1.0217881202697754, -0.6344172954559326, 1.0556976795196533, -2.500673294067383, -0.22993312776088715, 1.4991048574447632, -0.082855224609375, 1.5194593667984009, -0.8628737926483154, -1.0887562036514282, 0.3095643222332001, 2.0351169109344482, -0.44363144040107727, 1.3708630800247192, -0.6365965008735657, -0.08937318623065948, -1.997113585472107, -0.6476864218711853, -0.200927272439003, -0.08046065270900726, -0.6421114206314087, 0.6662729382514954, 0.9003927111625671, 0.23229549825191498, -1.2211624383926392, 0.7156292796134949, -0.2374534010887146, -0.4840165674686432, 0.8975580334663391, -0.5417255759239197, 0.8059635162353516, 0.5624622106552124, -0.5577380061149597, -0.597123920917511, 0.07821302115917206, 0.6425319910049438, 1.0894219875335693, 1.9903854131698608, -0.44431817531585693, -0.4247651994228363, 0.6558987498283386, -0.7899783253669739, -1.5779651403427124, -1.5245976448059082, 1.2626867294311523, -0.4765399396419525, -1.4205713272094727, 3.40213680267334, 0.03576502576470375, -1.1857271194458008, -0.2711097300052643, 0.27546244859695435, 0.12215346843004227, -0.061301495879888535, -0.0016432066913694143, 0.14494545757770538], [-0.7587614059448242, -3.270754098892212, 0.315290629863739, 1.4746665954589844, -0.03427634760737419, -0.9089350700378418, 0.8010904788970947, 0.4083165228366852, -0.09865757822990417, -0.4123860001564026, -0.31247037649154663, -0.1477399468421936, -1.3628913164138794, -1.273235559463501, 0.3081400990486145, 0.4954025447368622, 1.2228459119796753, -0.9012714624404907, -0.6775874495506287, 0.17011615633964539, -1.4976898431777954, 0.5898808240890503, -0.47240766882896423, 0.6614195108413696, -1.4533931016921997, -0.3830689489841461, 1.2200560569763184, 0.49051401019096375, -0.7936893105506897, -0.3248794972896576, -0.18203796446323395, -1.1487897634506226, -0.4784468114376068, 1.6587440967559814, 2.1490912437438965, 0.3205340802669525, 1.7389174699783325, -0.9080666899681091, -0.6695408821105957, -0.23863893747329712, -0.16601242125034332, 0.07083061337471008, -0.3360901474952698, -2.8095040321350098, -0.4614315927028656, 0.3799409866333008, -0.3801649510860443, 2.317883253097534, 0.7167826890945435, 2.11655592918396, 0.035227708518505096, 0.4054396450519562, 0.8178014755249023, 0.4202021360397339, 0.38754770159721375, 0.3388826549053192, -0.1912645399570465, 0.24346938729286194, 0.47742795944213867, -0.5673016309738159, 1.2688301801681519, 0.10341258347034454, 1.4032917022705078, 0.2794133126735687], [1.0362048149108887, 0.1994907259941101, 1.6892670392990112, 1.416542410850525, 0.5799198746681213, 0.8754300475120544, 1.3336310386657715, -0.17150790989398956, 0.10329057276248932, 0.4643890857696533, -1.6037697792053223, 0.5621350407600403, -1.246107816696167, -1.3478403091430664, -0.23000890016555786, 1.174858808517456, -0.3782493770122528, 0.8221912980079651, 0.4917314648628235, 0.7441723942756653, -0.9871259927749634, 1.8290642499923706, -0.2003745287656784, 1.6727869510650635, 1.3246026039123535, -1.6684051752090454, 1.5840256214141846, 0.6097977161407471, -0.2719632089138031, -2.1236555576324463, 0.20556257665157318, 1.7235468626022339, 1.7834779024124146, 0.17483091354370117, 0.3624413013458252, -0.7265908122062683, -0.8593868613243103, 0.3949602246284485, -0.034377340227365494, -0.563828706741333, 0.07932496070861816, -1.8482333421707153, 0.7395617365837097, -1.711368203163147, -0.8257768154144287, 0.05803174525499344, -0.5957471132278442, -0.2862071394920349, 0.8484262824058533, 0.1881568431854248, -0.46347543597221375, 1.4285348653793335, -0.2914767563343048, -0.1224946677684784, 1.0735834836959839, -0.5527810454368591, 0.8217986226081848, -0.14953632652759552, -0.792504608631134, -1.4024183750152588, -1.7415757179260254, -1.1071487665176392, 0.6451895833015442, 0.057065125554800034], [0.19694285094738007, 0.13892489671707153, -0.45611298084259033, 1.048232078552246, -0.3002912104129791, -1.8897464275360107, 0.4976207911968231, 1.5938831567764282, -0.511026918888092, 0.05311127379536629, 0.4762202501296997, -1.0035051107406616, -1.5130541324615479, 0.0968373715877533, 1.1918641328811646, 1.0324937105178833, 0.14964252710342407, -0.35476258397102356, 0.36548909544944763, 0.663496196269989, 0.8378938436508179, 1.9254740476608276, 0.16031251847743988, -0.8093727827072144, -0.0313536673784256, -1.2689343690872192, -0.452507883310318, 0.024820072576403618, 0.10033746808767319, -0.3772439956665039, 2.1203114986419678, -0.7059267163276672, 0.29813650250434875, 3.301218032836914, -0.3473554253578186, -0.965419590473175, -0.8751287460327148, -0.5524265170097351, 1.3717846870422363, 0.11077839136123657, -1.0816650390625, -0.014801363460719585, -0.6661702990531921, -2.8251309394836426, -1.5000079870224, -0.49018001556396484, -1.3968746662139893, 1.0163674354553223, 0.5672497749328613, -1.0354233980178833, 1.6273800134658813, -0.06541096419095993, 0.058502160012722015, 0.7898868322372437, 0.514746367931366, 1.2368056774139404, -0.3079526424407959, -0.24047088623046875, -0.5645952820777893, 1.1330580711364746, 1.2953085899353027, -0.8813452124595642, -1.4749739170074463, -0.4284403324127197], [-1.8269809484481812, -0.7752190232276917, -0.7877081632614136, -0.2429242730140686, 1.3021188974380493, 0.5061532855033875, -1.9028596878051758, -0.8706925511360168, -1.1506775617599487, -0.7814992666244507, -0.876182496547699, -0.09982061386108398, 0.5531965494155884, 0.12453705817461014, 2.388122797012329, -0.2989513874053955, 1.4603022336959839, 0.108418308198452, 0.614689826965332, -1.550601840019226, 0.7933903336524963, -0.9447524547576904, -0.22094237804412842, 0.3578072786331177, 0.5110464096069336, 0.94776850938797, 0.33799436688423157, -1.3084920644760132, -1.2348973751068115, -0.5989455580711365, 0.12922550737857819, -0.9192158579826355, -0.30615076422691345, -0.02176581509411335, -0.22226576507091522, 0.7563480734825134, -0.9665646553039551, -0.8482086658477783, -1.0848116874694824, 0.40309765934944153, -0.5755392909049988, -0.7531212568283081, 0.6658861041069031, 0.9303220510482788, -0.9677069187164307, 0.8136826157569885, -1.1772159337997437, 0.9351561665534973, -0.6261368989944458, 0.45656904578208923, 0.32223474979400635, 0.6912945508956909, -0.09769521653652191, -0.6550312638282776, -0.11870411038398743, 1.0231926441192627, 0.055565718561410904, 1.0506985187530518, 2.5530812740325928, 0.486669659614563, 1.1719759702682495, -0.13854365050792694, -0.22460435330867767, -0.3081284165382385], [0.5748893618583679, 2.580127716064453, 0.15786422789096832, -0.47260782122612, -0.4654850363731384, -1.1525804996490479, 0.8236756920814514, -0.2897619605064392, -2.0277459621429443, 0.4165995121002197, -2.0279812812805176, 0.49866798520088196, 1.0185493230819702, 0.9009718894958496, -0.7099928855895996, 1.5009628534317017, 1.5584468841552734, 1.0022714138031006, 0.27739131450653076, -0.6160847544670105, 1.5907323360443115, 0.033402834087610245, -2.170161724090576, 0.22198960185050964, 1.3880153894424438, -0.0667654275894165, 0.5059384107589722, -0.9658047556877136, -0.14620400965213776, 0.8458196520805359, 0.04451010003685951, 0.9066006541252136, 0.29185542464256287, 1.4075590372085571, -1.6141773462295532, -0.3844588100910187, -1.2576507329940796, 0.05397247150540352, -1.7946547269821167, -0.9724000096321106, -1.3522381782531738, -1.2754977941513062, 0.9088090062141418, -1.4626084566116333, -0.830580472946167, -0.9188363552093506, 0.5828355550765991, -0.9338961839675903, 0.35637611150741577, -0.6864566802978516, 0.6231784224510193, 0.7203592658042908, 0.3948807716369629, -1.5782581567764282, 2.150332450866699, -1.8364919424057007, 1.0611783266067505, -0.6753158569335938, 0.22028309106826782, -2.109758138656616, 0.6387073993682861, -0.9857572913169861, 0.37773948907852173, -0.10322072356939316], [-0.5050312280654907, -0.931115984916687, 0.32495859265327454, 0.9423999786376953, 0.948197603225708, 0.7708693742752075, -1.0908355712890625, 0.045737508684396744, -0.13059817254543304, -0.09128269553184509, -0.9958585500717163, 0.5776454210281372, -0.08533927798271179, 1.0314260721206665, -0.523131251335144, -1.2976020574569702, -0.17648188769817352, 0.9919944405555725, 1.1728651523590088, 0.9118407368659973, -1.440943717956543, 0.8948495388031006, 0.5951261520385742, -1.493712306022644, 2.197183132171631, 0.4016062915325165, 0.8013715147972107, -1.6084201335906982, -1.8102654218673706, 0.998558759689331, -1.948760986328125, 1.6144216060638428, 0.22744394838809967, -0.3672425448894501, -0.4730854332447052, -2.396540403366089, 0.5747091770172119, 0.4060499966144562, -0.6164315938949585, -0.6752961277961731, 0.4813607335090637, 0.8344125747680664, 0.45350515842437744, -0.6038952469825745, -0.27422818541526794, -1.5494898557662964, 0.6369500160217285, 0.8226313591003418, 0.2392919957637787, -0.49568241834640503, 0.6448569893836975, 0.007572907954454422, -0.46274951100349426, 0.3659100830554962, -0.01834886521100998, -1.8597979545593262, 0.4562714397907257, 0.8681386113166809, 2.118924617767334, -1.5074015855789185, -0.2033465951681137, 2.031400442123413, 1.2979285717010498, 1.255270004272461], [0.8788425326347351, -0.21727333962917328, -1.1047630310058594, 1.017033576965332, 1.8538641929626465, -1.5826679468154907, -0.5762273669242859, -0.6283579468727112, 1.1738486289978027, 0.732669472694397, -0.08395502716302872, -1.7727645635604858, 0.5538794994354248, -0.009599965997040272, 0.07362622022628784, 0.10390984266996384, -0.8311499953269958, 0.3897158205509186, -1.4023306369781494, -1.2718442678451538, -0.22653444111347198, -0.7878111600875854, -0.18792180716991425, -0.38708391785621643, -1.8081567287445068, 0.27811363339424133, 0.4196432828903198, -0.4165733754634857, -0.5833396315574646, 0.16255205869674683, -2.4509239196777344, -0.9808362126350403, 0.08063063770532608, 0.7040033340454102, 0.31805217266082764, -0.5264663696289062, -0.08922981470823288, -0.24771519005298615, -0.6513940095901489, -1.7293413877487183, -0.5231254696846008, 0.695734441280365, 1.8581676483154297, 1.2958873510360718, 1.118245244026184, 0.6816226243972778, 1.1886613368988037, -0.7781417369842529, -0.871356725692749, -0.950168788433075, -0.5235936045646667, 0.2456110268831253, 1.5696874856948853, -0.2514328062534332, 0.47694361209869385, 0.7160711884498596, 0.588817834854126, -0.6540811061859131, -0.5229694247245789, -0.20674927532672882, 1.4171677827835083, 0.7081791758537292, 1.0654453039169312, -2.078691244125366], [-0.9820318818092346, -0.16619493067264557, -0.9217976927757263, 1.7951905727386475, 0.008492608554661274, 1.0272178649902344, 1.1513735055923462, 0.7050440311431885, -0.18212270736694336, 2.251204013824463, 1.0635087490081787, -0.7478629946708679, 0.45079779624938965, -0.3197832405567169, -0.22775350511074066, -0.9876241087913513, 0.35367682576179504, 0.9690803289413452, 0.9444662928581238, 0.895927369594574, 0.4887847304344177, -0.22457295656204224, -0.24945564568042755, 0.12242203950881958, 1.2026203870773315, 0.008830060251057148, -1.4005677700042725, -0.21877598762512207, 0.14879000186920166, 0.36465877294540405, -0.6594139933586121, 2.3213002681732178, 1.8034075498580933, 2.1965484619140625, 3.0451667308807373, -0.6008340716362, 0.7828478217124939, 0.5556546449661255, 0.45609113574028015, -1.0756494998931885, 0.641947865486145, 0.05725952237844467, -0.15489616990089417, 1.168274164199829, -0.45033949613571167, -0.8926440477371216, -0.7996522784233093, -0.2793235778808594, 0.014578177593648434, -1.7687090635299683, 0.32917675375938416, -0.8201868534088135, 0.2541660964488983, 1.1063933372497559, -0.48605623841285706, -1.1237404346466064, 0.7510207295417786, 0.8453274369239807, -1.442760705947876, 0.1155681312084198, -1.9632554054260254, 0.07674958556890488, 0.2166961282491684, -0.03497489541769028], [0.8295544385910034, 0.34188660979270935, 2.0281424522399902, -0.463329941034317, 1.11358642578125, -0.07998841255903244, -0.6604186296463013, -1.563929796218872, -1.5426875352859497, -1.149430751800537, 0.22396299242973328, 1.2324838638305664, 0.15689446032047272, 2.541818141937256, 0.045791175216436386, 1.1514819860458374, 1.5108176469802856, -0.44582754373550415, -0.9105081558227539, 0.2575641870498657, 0.8059789538383484, -0.8045524954795837, 0.3470369875431061, -1.7725824117660522, -0.37845322489738464, -0.0967993214726448, -2.583015203475952, 0.24360188841819763, -1.3746414184570312, -1.228993535041809, -0.24504277110099792, 0.07307657599449158, -0.35079100728034973, 1.097062587738037, -0.8765669465065002, -1.096881628036499, -0.14904844760894775, -1.692684292793274, -0.1409318745136261, -0.017669422551989555, 0.3253997266292572, -1.7458980083465576, -1.1907775402069092, 0.4100185036659241, 0.23750361800193787, -1.9000407457351685, -0.5624759793281555, 0.17592625319957733, 0.5525745153427124, 1.1884006261825562, 0.38041460514068604, -0.43464159965515137, 0.5943290591239929, 0.2964452803134918, -0.4159003794193268, 2.145841598510742, 0.40717485547065735, 0.12790338695049286, -0.092875637114048, 0.7190704345703125, 0.008091281168162823, 2.5661513805389404, 0.3670043349266052, -1.336911916732788], [1.633608102798462, 0.07762494683265686, 2.2887685298919678, 0.8930908441543579, 0.7760353684425354, 0.040377940982580185, 1.3807430267333984, -0.3374449610710144, -0.46399596333503723, 1.6209373474121094, 1.1953877210617065, 0.16639293730258942, 0.5706443190574646, -1.5303972959518433, 0.8398000597953796, -0.19510148465633392, 2.360812187194824, 0.3955751359462738, 0.41863927245140076, -0.1981484442949295, -0.5810753107070923, -0.20839941501617432, 1.0438040494918823, 0.1328294426202774, 0.543872058391571, 0.6241143941879272, 0.27011731266975403, 0.07816886156797409, 2.176010847091675, 0.25067606568336487, 0.9869764447212219, -0.6042077541351318, 0.48573657870292664, -0.524516761302948, 1.2549952268600464, -0.180280864238739, 1.82419753074646, 0.2896621525287628, -0.4013613164424896, 0.4992288649082184, -1.0574239492416382, 0.7374563813209534, 0.9467185139656067, -1.0054537057876587, 1.4758186340332031, -0.05570260435342789, -0.4073420763015747, 0.29103174805641174, -1.416076898574829, 0.0003729707677848637, -0.03500400856137276, 0.18590223789215088, 1.4111523628234863, 0.09383002668619156, -0.10224856436252594, 0.6958387494087219, 0.859653651714325, -0.09252525866031647, -0.39694085717201233, 0.1025710478425026, -1.2406340837478638, 0.24328583478927612, -1.7055925130844116, 0.41178953647613525], [-0.3649979531764984, -0.046247027814388275, 0.13773541152477264, -0.4042549729347229, 0.2629477083683014, -2.6076557636260986, 2.212329864501953, -1.4964594841003418, 1.2127854824066162, 1.187469244003296, 0.47250065207481384, -0.06607022136449814, -0.12608154118061066, -0.2793864607810974, -0.0823250263929367, 0.705518364906311, 0.26939845085144043, -0.7586264610290527, 0.5570112466812134, 1.2007569074630737, -0.8118877410888672, -0.2354823499917984, -0.25239095091819763, -1.3957115411758423, 1.8943215608596802, -1.7234694957733154, 1.6683781147003174, 0.17668239772319794, -1.637961506843567, 1.159458875656128, 0.4525337219238281, 1.8876944780349731, 0.48073092103004456, -0.1889723390340805, -0.4785398244857788, -0.5856767296791077, -1.0045372247695923, 0.30841708183288574, -1.6936668157577515, 3.016230821609497, -1.3252605199813843, 1.535416841506958, -1.2576192617416382, -1.2619754076004028, 0.5255542397499084, 0.09982074797153473, 0.09027399867773056, 0.7776064872741699, 1.644843578338623, 0.2179257720708847, 0.7860738039016724, 0.6209539175033569, -0.3269329071044922, 0.6456055045127869, -0.27355873584747314, -0.5819429159164429, -1.0754081010818481, 0.5547609329223633, -1.6246116161346436, 0.9756993055343628, -0.4640841484069824, 0.24687525629997253, -0.3035045862197876, 0.9863592982292175], [-0.7007573843002319, 0.3809182047843933, -1.7391445636749268, 0.5171976089477539, -0.8189684152603149, -0.7855912446975708, -0.2647463083267212, -0.6128501892089844, 1.6316590309143066, 1.410699486732483, 1.1388989686965942, -0.6812098026275635, -0.7189850807189941, -0.1308617889881134, 0.7365513443946838, -2.1979269981384277, -0.6174795031547546, -1.1330653429031372, -0.8584293127059937, -1.2657886743545532, -1.4672852754592896, -0.17408375442028046, -1.1674072742462158, -0.4620151221752167, -0.45200300216674805, 0.32998931407928467, -1.9764379262924194, 0.7845156788825989, 0.24917925894260406, -0.3206409811973572, 1.0425487756729126, 0.985808253288269, 0.9965298771858215, 0.4795092046260834, -1.3554562330245972, -0.8842463493347168, -0.8878602385520935, 0.3674176335334778, -1.4195716381072998, -0.4321667551994324, 1.3858073949813843, 0.6360766291618347, 1.6392226219177246, -0.4510013163089752, -1.8765122890472412, 2.236936330795288, -0.3779864013195038, 0.6171927452087402, -0.5748094320297241, 0.07338227331638336, 1.554182767868042, -1.810971736907959, 1.8614305257797241, 0.0644024908542633, 0.25968703627586365, 0.8631811141967773, 1.1242566108703613, -0.33794716000556946, -0.5209648609161377, 0.13308849930763245, -0.42583805322647095, -0.3075569272041321, -0.9217967391014099, 0.6489348411560059], [1.614890217781067, -0.21274854242801666, -1.3791528940200806, -1.4907453060150146, 0.095681332051754, -1.8666496276855469, -0.4923310875892639, -0.022331442683935165, 0.5300057530403137, 0.11115887761116028, -0.4732258915901184, 0.805683434009552, 1.4041433334350586, -0.5178985595703125, 1.0801507234573364, -0.2555212378501892, 0.7638908624649048, 1.1631439924240112, 0.5877048969268799, -0.6912453770637512, 0.5152939558029175, 0.4129524528980255, 0.16694115102291107, -0.050300367176532745, -0.873750627040863, -1.3881101608276367, -0.6597690582275391, -1.4513126611709595, 0.31675827503204346, 0.9152454137802124, -0.194498673081398, -0.9215844869613647, 0.247441366314888, 0.37950778007507324, -0.40596312284469604, -2.6350698471069336, 0.43506693840026855, -0.5263382196426392, 0.6532183289527893, 0.6035804748535156, -0.06992097198963165, 0.05660098418593407, 1.0727649927139282, -1.7983211278915405, 0.021652793511748314, 0.2545943856239319, 0.8612858057022095, -0.6431301832199097, -1.01374089717865, 1.212380290031433, 1.6119401454925537, -0.12499216198921204, 0.1646171659231186, -1.1099450588226318, -0.35669392347335815, -0.00021206984820310026, -0.0024134726263582706, 1.5928528308868408, -0.7080791592597961, -0.6302135586738586, -1.555959939956665, 1.1434757709503174, 1.2408500909805298, -0.1843404620885849], [1.6888375282287598, 0.3406928479671478, 1.6173707246780396, 0.3910984694957733, 1.1258714199066162, -1.0886704921722412, -1.1666299104690552, 0.779992938041687, 0.20462338626384735, -0.5097662806510925, 0.34819209575653076, 0.07894232869148254, 0.3393813371658325, -0.3016339838504791, 0.04618155583739281, 0.3054671287536621, 1.7538390159606934, 0.4081652760505676, 0.4728831648826599, -1.6847939491271973, 0.3570675253868103, -1.3562570810317993, 2.072458028793335, -0.021897321566939354, 0.8362343907356262, 0.5537328124046326, 0.3752906620502472, 0.012628378346562386, 0.2530479431152344, 0.44817429780960083, 0.18095692992210388, -0.22810626029968262, -0.45764851570129395, 0.502438485622406, 0.16578806936740875, 0.16171866655349731, 0.6897887587547302, -0.17976810038089752, -1.0768005847930908, -0.14522026479244232, 0.5686219334602356, -0.5966563820838928, -1.4153342247009277, 0.786689817905426, 1.3260657787322998, -0.29605957865715027, 0.32805392146110535, -2.296387195587158, -1.3473820686340332, 0.18431949615478516, 0.3736148476600647, -1.7995846271514893, -0.1791294366121292, -0.49360039830207825, -1.774311900138855, 0.8376073241233826, -0.40925800800323486, 0.7046855688095093, 0.28092771768569946, -0.8322222828865051, 0.19302305579185486, -0.6677277088165283, 0.04541275277733803, 0.17004717886447906], [1.3851606845855713, 0.3356809616088867, 0.8348387479782104, 1.4977483749389648, 0.5952630043029785, -0.09286405146121979, -0.821999192237854, 1.9499871730804443, -0.4301108717918396, 1.6681725978851318, -1.0005877017974854, -0.6935067772865295, 0.3990292251110077, -1.2655566930770874, 0.7399025559425354, -2.7237961292266846, -1.3532400131225586, 1.5580822229385376, 0.10534340888261795, 0.035015784204006195, 1.1700994968414307, -0.7680520415306091, -0.5868845582008362, 0.8297072649002075, -1.12967848777771, 0.6255642771720886, 0.5793235301971436, 2.1242523193359375, 1.2086386680603027, -0.14249666035175323, 0.09622984379529953, -0.02659817785024643, -2.5340311527252197, 0.8813972473144531, 1.120969295501709, 1.114638090133667, -1.3241009712219238, 1.284085988998413, -1.9947189092636108, 0.10548251122236252, -0.3826228976249695, 0.3231956958770752, 1.3362631797790527, 0.06828352063894272, 1.871525526046753, 1.2188901901245117, -0.01350014004856348, 0.5323542356491089, 0.9219546318054199, -0.8036515712738037, -1.0570200681686401, -0.842727541923523, -0.1305617392063141, 1.273738145828247, 1.4921549558639526, 0.9379393458366394, -0.32299676537513733, 0.36291027069091797, 1.1475985050201416, -0.2839992046356201, -1.4234106540679932, -1.0563867092132568, -0.3042234182357788, -0.8926429152488708], [-0.7295787334442139, 1.5515975952148438, 0.6704548001289368, 0.44373878836631775, -1.2399489879608154, -1.05833899974823, -1.0970044136047363, 2.8490161895751953, -0.6569908261299133, -0.5143629312515259, -0.5948000550270081, -0.5485944747924805, 0.2145422399044037, -2.011941432952881, 0.0549338273704052, 0.14417630434036255, 1.2211761474609375, 1.0414206981658936, -2.954460382461548, -0.6065916419029236, 0.2177727222442627, -1.3721027374267578, 0.5053967833518982, -0.01320742815732956, 0.01951046846807003, 0.8530362844467163, 0.5381094813346863, -0.34833481907844543, 0.8446069955825806, 0.17052684724330902, 0.5177038908004761, -0.5967574715614319, 0.9610818028450012, 0.04710691049695015, -0.07458171248435974, -1.99187171459198, -0.09803148359060287, -1.2809858322143555, -0.4442022442817688, 0.8944735527038574, 0.6489201784133911, -0.03801967576146126, -1.199215054512024, 0.4476432204246521, 0.2904433012008667, -0.17930403351783752, 2.3714194297790527, -0.44243741035461426, -0.5130928754806519, 0.019765162840485573, -0.4211438000202179, 0.19516047835350037, 1.2922799587249756, 0.009069671854376793, -0.6824706792831421, 1.3192352056503296, -0.48512786626815796, -0.8166964650154114, -0.42671892046928406, 1.1635730266571045, -1.2860177755355835, -1.0779922008514404, -0.739214301109314, -1.224254846572876], [-0.9190008044242859, -1.660977840423584, -0.10059656202793121, -1.8939982652664185, 0.09136415272951126, -0.20923537015914917, 0.12298117578029633, 0.8076676726341248, -0.3215887248516083, 0.6434090733528137, -0.4963209927082062, 0.6575766801834106, -0.13268420100212097, -0.4919907748699188, -1.0451748371124268, -1.7630666494369507, -1.8198193311691284, 2.4876227378845215, 0.9426286816596985, 0.3123924434185028, 0.6289490461349487, 0.318569153547287, -1.087869644165039, -1.2019367218017578, 1.1400545835494995, 0.9789938926696777, -0.9737967848777771, 0.6086476445198059, 0.607197105884552, -1.4109163284301758, -0.28574976325035095, -0.7991772890090942, -1.2455354928970337, 0.837088942527771, -1.1787225008010864, -0.39069026708602905, 0.2782847285270691, 0.9447006583213806, -0.49256739020347595, 0.7470368146896362, 0.8595454692840576, 0.9378110766410828, -1.0980935096740723, 1.2322888374328613, 2.3002583980560303, 1.7804176807403564, 0.2732095718383789, -0.19315358996391296, -0.8454523682594299, 0.2614462077617645, 1.9754762649536133, -0.5866787433624268, -1.221368432044983, -0.5090452432632446, 0.24983569979667664, 1.2602537870407104, -0.8025563359260559, 1.8099206686019897, -1.7626999616622925, -0.26425638794898987, -1.0264824628829956, -0.9662203788757324, -2.4180591106414795, -0.9794816970825195], [0.958528459072113, -1.0094159841537476, -0.7538266181945801, -1.115050196647644, -0.05311547964811325, -1.9539685249328613, -0.12497501820325851, -0.24525372684001923, -0.7526636123657227, -1.139013409614563, 1.555469274520874, 0.8257489204406738, 1.7372421026229858, 1.2108420133590698, -1.5374104976654053, 0.4552355408668518, 0.2619049549102783, -0.6104686260223389, 1.3483151197433472, -0.395866721868515, -0.21041536331176758, 0.4291647970676422, 0.4284408688545227, 1.8465720415115356, 1.625586748123169, 0.18809404969215393, 0.46733030676841736, -0.42897170782089233, -0.7107564210891724, -0.5411134362220764, 0.6773747801780701, 0.25320619344711304, 1.0225759744644165, 0.5103684067726135, -1.2303698062896729, -0.9195134043693542, 0.10595784336328506, 0.05382524058222771, 0.009056171402335167, -0.46765610575675964, -0.4280499517917633, -0.3244043290615082, 0.13628554344177246, -0.19327372312545776, -0.021204998716711998, -0.8864176273345947, 0.2671101689338684, 1.9884313344955444, -0.6860391497612, -0.1832031011581421, -0.3624163568019867, -0.18747086822986603, 1.564109444618225, -0.49270227551460266, -0.7024623155593872, -2.2420194149017334, -1.869019627571106, -0.27215075492858887, -0.7275820970535278, -0.7215858101844788, 0.42841410636901855, 0.5828791260719299, -2.0842113494873047, 0.4473901689052582], [2.3553848266601562, -0.812617301940918, -2.3592488765716553, 0.5078686475753784, -0.17768369615077972, -1.335012435913086, 0.510523796081543, 0.29288968443870544, 0.28183600306510925, -0.06164152920246124, 0.9568212628364563, -1.5124987363815308, -1.2262961864471436, -0.3747766613960266, 0.3183175027370453, 0.5913106799125671, 0.734532356262207, -0.9286635518074036, 0.8543026447296143, -0.5317947268486023, -1.505321979522705, 2.432940721511841, -0.8977674841880798, 0.5573731064796448, -0.19995540380477905, 0.672290027141571, -0.3001524806022644, -0.5787186026573181, -0.30527016520500183, 0.6944347620010376, -3.2327113151550293, -1.5904661417007446, -1.0915961265563965, -1.153694748878479, -0.5303020477294922, 1.762154459953308, -0.49617668986320496, 0.7452744841575623, -1.2407478094100952, -1.5792146921157837, -1.487825632095337, -0.7391999363899231, 0.6636130213737488, 0.30359765887260437, -0.11664137989282608, -0.7021650671958923, 0.06388465315103531, -0.2314893901348114, 0.6347474455833435, 2.180922746658325, -0.9122455716133118, -1.2506561279296875, 0.38143593072891235, -0.6361752152442932, -1.93776273727417, 0.8982729315757751, -1.5357112884521484, 0.44211381673812866, 1.2549272775650024, -1.3871665000915527, 0.14420579373836517, 0.9411237835884094, -0.422122985124588, -1.7384041547775269], [-0.5119736790657043, 0.4553660750389099, -1.0733773708343506, -0.21806614100933075, -0.20875805616378784, -0.21641817688941956, 0.1197192445397377, 0.003735469887033105, 1.04939603805542, 0.4425383508205414, -0.6182726621627808, -1.6976244449615479, -0.1275988072156906, 0.5535076856613159, -0.24571847915649414, 1.2772786617279053, 1.6075857877731323, -0.0527159608900547, 0.6688588261604309, -1.4763611555099487, 0.5997114181518555, 0.7563565969467163, -0.28312090039253235, -0.7326453924179077, 0.29834118485450745, -1.676944613456726, -1.5338460206985474, -0.8831788301467896, 1.487594485282898, -0.6782258749008179, -0.4967702329158783, -2.4606332778930664, 0.4052114188671112, 1.0464160442352295, 0.5703309178352356, 0.4881662428379059, -0.38067305088043213, 0.08825132250785828, 0.02663254179060459, 0.2501843571662903, -0.595818042755127, 0.8409401774406433, -1.0818735361099243, -1.2620903253555298, -0.007654835004359484, -0.2811385989189148, -1.9199838638305664, 0.8969739675521851, 0.6805904507637024, 0.11703147739171982, -0.7216029167175293, -0.2371101826429367, -0.30558186769485474, 0.8299200534820557, 2.2680811882019043, 1.445684552192688, -1.3276265859603882, 0.8872137069702148, -0.6702201962471008, -0.21230383217334747, 0.01234354730695486, -0.8975613117218018, 0.27063387632369995, 0.7037994861602783], [-0.872063398361206, -2.2286217212677, -1.095601201057434, -0.6462503671646118, -0.4057062268257141, -0.18625913560390472, -0.04498116672039032, -1.8602417707443237, -1.801321268081665, 0.8956629037857056, -0.3335845470428467, -0.15379206836223602, -2.1100399494171143, -0.25569459795951843, 0.30075788497924805, -1.393865942955017, -0.4283599853515625, 0.9038400053977966, 0.49803826212882996, 1.2896791696548462, -1.0227926969528198, 0.10785841196775436, 0.12152814865112305, -0.12144157290458679, -1.3275002241134644, -0.007612226530909538, 0.45598992705345154, 1.5481830835342407, 0.4065331220626831, 0.3752884268760681, 0.4646259546279907, -1.2933238744735718, -0.7657520174980164, -0.11960677057504654, -0.10541099309921265, 0.9006471037864685, 1.294067144393921, 0.7483006715774536, -1.0132331848144531, 0.681587815284729, 0.10631061345338821, 2.111137866973877, -0.9586313962936401, -0.17909806966781616, -0.19983823597431183, 0.3642892837524414, 3.122087240219116, 0.8753520250320435, -0.6886174082756042, -0.934766948223114, 0.10640635341405869, -0.3587496876716614, 0.8262677192687988, 0.07785025238990784, -1.9134057760238647, -0.49285954236984253, -1.2161030769348145, -1.3003123998641968, -0.5738027095794678, -1.1380382776260376, 0.7963230609893799, 0.5081366896629333, 0.587754487991333, 0.4395512342453003], [-0.40283069014549255, -1.9152593612670898, 0.7020952105522156, 1.068199634552002, 3.5177173614501953, -0.2900148928165436, -1.2733221054077148, -0.2339077889919281, 0.3533449172973633, 0.4126202464103699, 0.4733986556529999, -2.6394641399383545, 0.004900556057691574, 0.2153186798095703, -1.7343177795410156, 1.168364405632019, 0.5378002524375916, -1.3963762521743774, 0.41960322856903076, 0.2214193195104599, -0.07429295033216476, -0.4447878897190094, 0.04963169991970062, 1.4741991758346558, 1.7238765954971313, -0.5749472975730896, 0.5453272461891174, 0.25883248448371887, -0.6330980062484741, -0.012113062664866447, -0.7841876745223999, 0.79252690076828, 0.3295680284500122, 1.4873930215835571, -1.046878457069397, -0.9063586592674255, 0.3597361743450165, 1.8732467889785767, -0.6135297417640686, 0.8003865480422974, -0.9419698119163513, 0.17005260288715363, -0.7297120094299316, 1.1571961641311646, -0.2130574882030487, 2.2366113662719727, -0.22429011762142181, 0.05826137587428093, 0.6032696962356567, 0.737610936164856, 0.023678161203861237, 0.972224771976471, -0.302313894033432, 1.421953797340393, 1.8176722526550293, 0.44287043809890747, 0.837379515171051, 0.6210570931434631, 0.43464797735214233, 1.109706163406372, 0.9987831115722656, -0.42715421319007874, 0.8094421625137329, -0.5584933757781982], [-0.3444012403488159, 1.2625985145568848, -0.22851574420928955, 3.3533153533935547, 0.25791388750076294, 1.8098292350769043, 1.9804058074951172, 1.349736213684082, -2.709836959838867, 0.3889657258987427, 0.7192005515098572, -0.5830459594726562, 0.2519100308418274, -1.095241904258728, -0.8272406458854675, -0.31125229597091675, 0.6086130142211914, 0.47367092967033386, 0.8925759792327881, 0.18591350317001343, -0.2076430469751358, 1.6804418563842773, -1.4046761989593506, -1.0824424028396606, 0.6327160000801086, -0.46594369411468506, 0.8577221035957336, -0.13132798671722412, -1.4972450733184814, -1.4005911350250244, 2.2518820762634277, -0.2817205786705017, -1.177436113357544, 1.524453043937683, -0.13769184052944183, 0.6757864952087402, 0.6153462529182434, 2.0712368488311768, -0.2564544975757599, 0.08550907671451569, -0.7323039174079895, 0.47230979800224304, 0.8004911541938782, -1.8660967350006104, -0.8133909106254578, -1.0809749364852905, -1.7695503234863281, -1.7899523973464966, -0.7493121027946472, -0.30875110626220703, -0.11729686707258224, -1.4338223934173584, 0.07552246749401093, -0.8476037383079529, -0.9551360011100769, 1.3502120971679688, 1.9274556636810303, -1.3906346559524536, -0.8113964796066284, 1.080471158027649, 0.020465871319174767, -0.3035675883293152, 1.8815854787826538, -1.4960227012634277], [1.7367695569992065, 0.007938672788441181, -0.643974781036377, -0.3392929434776306, -1.2227755784988403, 2.2029385566711426, -0.6631407737731934, -0.3651660978794098, 1.5942052602767944, 0.03301534429192543, 0.2396155595779419, 0.329111784696579, -1.0362803936004639, -1.0773673057556152, 0.36482805013656616, -0.1668112725019455, 0.2847420871257782, -1.3300225734710693, 0.5754587650299072, -1.1092865467071533, -0.05720173567533493, -0.2297602742910385, 0.04396497830748558, 0.16208651661872864, -2.2064573764801025, -0.11610688269138336, 0.4399507939815521, 0.8020865321159363, 0.9970673322677612, 0.20520739257335663, 1.0621064901351929, 0.41739532351493835, -0.36856403946876526, -0.4649695158004761, 1.6097007989883423, 0.7573699951171875, -0.6495270729064941, -1.7016282081604004, -1.0417953729629517, 1.4746088981628418, 2.3102824687957764, 0.8288062214851379, 1.8915098905563354, 0.9280052781105042, 0.3563125431537628, 0.10016395151615143, -0.047906044870615005, 0.37373873591423035, -0.5791043043136597, 0.4259275496006012, 0.6298475861549377, -1.0426931381225586, -1.597225308418274, 0.9712472558021545, 0.4146302342414856, 1.3378264904022217, -1.3152742385864258, -1.3468544483184814, 1.0622880458831787, -0.9511858224868774, -0.1644531935453415, -0.5017624497413635, 0.3467335104942322, 0.9883401393890381], [1.0531020164489746, -1.5232725143432617, 0.45906707644462585, -0.8516513705253601, -0.6183772087097168, 0.39298224449157715, 0.11479496210813522, -1.3253002166748047, -1.0399712324142456, -0.7844459414482117, -0.23616738617420197, -0.0004047650145366788, -0.04882798716425896, -1.078595757484436, -0.3170485496520996, -0.10611722618341446, -0.9357528686523438, -1.3505284786224365, 0.4537959396839142, 0.7893531322479248, 2.3267133235931396, -0.16909900307655334, -0.31686723232269287, -1.2961324453353882, -1.524755597114563, -0.4943084716796875, 0.49016857147216797, 0.9610323905944824, -1.2078334093093872, -0.3019724190235138, -0.0336875356733799, -0.6692810654640198, 0.41293013095855713, -0.9860551357269287, 1.524057149887085, -0.717690646648407, 1.2494288682937622, -1.0027121305465698, -0.4788777232170105, -1.3879913091659546, 1.3616833686828613, 1.5017811059951782, -1.0282444953918457, 0.03130227327346802, -0.38188910484313965, 0.2042652815580368, -0.2773222327232361, -1.0442720651626587, 0.5233882665634155, 0.32350045442581177, 0.366393119096756, 1.0218192338943481, -1.2651623487472534, -0.33023932576179504, -0.5998042225837708, 0.5953306555747986, -0.5388776659965515, 0.28512656688690186, -0.8406445384025574, -0.7807870507240295, -0.8802384734153748, -0.35578224062919617, 0.575055718421936, 1.0404648780822754], [0.15315639972686768, 0.04807685315608978, -1.0435998439788818, 0.17506656050682068, -0.5397564172744751, 0.7156438827514648, -0.13512474298477173, -1.3085339069366455, -1.513061761856079, 0.6951537728309631, 0.7677117586135864, -1.0103236436843872, -0.2800288498401642, 0.7484668493270874, -1.9375673532485962, 0.7523590326309204, -0.46485093235969543, 0.5548238158226013, 1.1345016956329346, -0.26261061429977417, -0.16846013069152832, -0.06197885051369667, -0.1706133931875229, -1.405509352684021, 0.27513378858566284, 1.2980308532714844, -0.7934295535087585, -0.14346933364868164, -1.0454634428024292, -0.16907694935798645, -0.4458962082862854, -0.5272789597511292, 0.7703887224197388, -0.14066295325756073, 1.247819185256958, -1.7671211957931519, 0.7917287945747375, -2.0696678161621094, -0.3436598479747772, 1.2196686267852783, -0.4718228280544281, 0.32516440749168396, -0.7148684859275818, 0.29109328985214233, 0.040557969361543655, -0.7439899444580078, 0.04164281487464905, -2.551452875137329, 0.5459828972816467, 0.2748074531555176, 0.25444355607032776, -0.968251645565033, -1.6733125448226929, -1.2775177955627441, 0.39444321393966675, 0.47577622532844543, 0.8319456577301025, -0.10742581635713577, -0.32557255029678345, -0.7484260201454163, 1.1606405973434448, -0.5209614038467407, 0.6627086997032166, 0.08465953916311264], [-0.4296579360961914, -0.7447468638420105, -0.10649638622999191, 0.2660881578922272, -1.2707791328430176, -0.8931336998939514, 0.9020151495933533, -0.23728875815868378, 0.06644044816493988, -0.056941717863082886, 1.0887763500213623, -1.2388296127319336, -0.09512779116630554, 0.8395275473594666, 0.31228581070899963, -0.6638588309288025, -0.549494206905365, 0.21675236523151398, 0.27664440870285034, -0.4428626298904419, 0.9058521389961243, 1.1381176710128784, 0.5447078943252563, 0.04332819953560829, -0.00798873696476221, -0.7946362495422363, -0.44611090421676636, 0.17641980946063995, -1.6608707904815674, 0.20260609686374664, -0.767888069152832, -0.5266494750976562, 0.28598552942276, -0.07648008316755295, 1.0274358987808228, -1.0316261053085327, 0.18180467188358307, 1.3967983722686768, 0.9021516442298889, 0.4382324814796448, 0.6658279895782471, -0.2751321494579315, 0.19478794932365417, 0.7656012773513794, -0.6608772873878479, 0.716966986656189, 0.10690529644489288, -0.15237221121788025, 0.22276277840137482, 0.37575581669807434, -0.7988558411598206, -1.4028046131134033, 0.9279139041900635, -1.3631736040115356, -1.0152734518051147, 0.6633597016334534, -0.7425881624221802, 1.698797583580017, -1.1020286083221436, -0.06914452463388443, -1.2281413078308105, 1.0546923875808716, -0.41142019629478455, -0.043990399688482285], [-0.8730295300483704, 1.1584539413452148, -1.2544825077056885, -0.8004186749458313, 0.4425507187843323, -1.597329020500183, -1.2520197629928589, -0.4223710298538208, -0.518329381942749, -1.4176609516143799, -1.146316409111023, -0.2736819088459015, 0.8841924667358398, 0.922213077545166, -0.7871072292327881, 0.4971476197242737, 1.139854073524475, 0.4472878575325012, 0.19973088800907135, 1.0123417377471924, -0.7237678170204163, -2.5820624828338623, -0.18985895812511444, 1.8758926391601562, 0.004295577295124531, 0.2782823145389557, -0.08690748363733292, 0.05077787861227989, 1.8880094289779663, 0.04339859262108803, 0.6204605102539062, -1.3153836727142334, -0.49255481362342834, 0.8446352481842041, 0.35378897190093994, 0.2695082128047943, 0.1347476691007614, -0.7308133840560913, 1.4624788761138916, 0.5206689834594727, -0.8837918639183044, 0.584409236907959, 0.3243039548397064, -0.025708960369229317, -0.6733405590057373, 0.41059818863868713, -0.4236368238925934, -0.30880358815193176, 0.8572942614555359, -0.8865442872047424, -1.598944067955017, -0.9626750349998474, -0.3715217709541321, 0.5526748895645142, -0.2848701775074005, 1.3374346494674683, 0.9192628264427185, 0.03836548328399658, -1.264035940170288, -0.4978761076927185, -0.3541312515735626, 0.5848968625068665, -0.12679322063922882, 0.02014928124845028], [-0.4130190312862396, -1.2359282970428467, 1.045688271522522, 0.6595064997673035, 0.7094871401786804, -0.8833624720573425, 0.8754067420959473, -0.11184186488389969, -0.8040903210639954, 0.4582486152648926, -1.8516387939453125, 0.2457779496908188, 0.1817358285188675, -0.5299777984619141, 0.050292253494262695, -0.46829208731651306, 0.2511693835258484, -0.9157454371452332, 1.9794502258300781, 1.1345791816711426, 0.7358406782150269, -0.2892864942550659, -0.8397499322891235, 0.9446876049041748, 0.7169291377067566, 1.1587647199630737, -0.5607540011405945, -0.893575131893158, -0.5379098057746887, 0.3330017030239105, -0.10830461233854294, -0.3863506019115448, 0.26585379242897034, -1.0432484149932861, -0.7251838445663452, -0.18477630615234375, 0.7321299314498901, -0.9162538647651672, 1.9150656461715698, 1.6829086542129517, -1.0009853839874268, -0.4050012230873108, 0.07508186250925064, 0.7896119952201843, 1.5240517854690552, 1.1901731491088867, 1.4947166442871094, 1.2096835374832153, 0.11855155229568481, -0.3904065787792206, -0.4181225299835205, -0.46553879976272583, -0.0927458181977272, -0.07967172563076019, -0.5719033479690552, 0.29780226945877075, -0.7178995609283447, 1.1037999391555786, -0.7955867052078247, -1.3642141819000244, -0.3516496419906616, -0.12097665667533875, 0.8701377511024475, 0.22200776636600494], [-0.596442699432373, -0.7315966486930847, 1.3444331884384155, -1.2797476053237915, -0.4929535984992981, -0.07152952998876572, 0.04896144941449165, 1.6260778903961182, 1.1749095916748047, -0.6297655701637268, -1.5971314907073975, 0.10747021436691284, 2.3347482681274414, -0.02225309982895851, -1.4298630952835083, 0.07521161437034607, -0.43254372477531433, 1.1512248516082764, 0.06646114587783813, -0.11359473317861557, 1.081313967704773, 0.5568195581436157, -1.056152582168579, 0.4899657070636749, 0.9546833038330078, 1.0908408164978027, -0.7098596096038818, -0.9241841435432434, -0.9220011830329895, 1.1127312183380127, 0.08349219709634781, -1.2707325220108032, 0.39023616909980774, 2.4342823028564453, 0.9079708456993103, -0.720514714717865, 0.23134110867977142, -0.9903495907783508, 0.27312958240509033, -1.2044578790664673, 0.30394819378852844, -1.230650544166565, -1.6722462177276611, 0.8007797598838806, -0.304517924785614, 0.026714017614722252, -0.881532609462738, -0.5986961722373962, -0.7604767680168152, -0.472320020198822, 1.7102227210998535, -0.41278910636901855, -0.06083350628614426, -1.3372613191604614, 0.1284891963005066, 0.8818211555480957, -0.27413007616996765, -1.1756625175476074, -0.6303789615631104, -0.0284982118755579, -1.8816261291503906, 1.0327072143554688, -1.0446429252624512, -1.686959981918335], [0.3563102185726166, 0.8824110627174377, 0.14727655053138733, 1.3784093856811523, 1.5799884796142578, 1.091314673423767, -1.583039402961731, -0.1063089668750763, 0.9109355211257935, 0.8288953304290771, 1.7721165418624878, -0.3320617973804474, -0.08088002353906631, -0.5810355544090271, -0.2180088460445404, -0.06052509695291519, 0.44231799244880676, -0.8125333786010742, -2.010373592376709, 0.32994234561920166, 1.4557217359542847, -1.2518129348754883, -0.013186217285692692, 0.3045894503593445, 0.11950083822011948, 1.569298505783081, 0.4523305594921112, 0.6955327391624451, -0.6745948791503906, -0.660936176776886, -0.6927865147590637, -1.3785152435302734, 0.9738855361938477, 0.13572710752487183, 0.9418838620185852, -0.3190089166164398, -0.9959372878074646, -1.295789122581482, 2.78997802734375, -0.9486463069915771, -1.4167134761810303, -0.11788804829120636, 1.0232139825820923, -2.4991211891174316, 1.1347935199737549, 0.9518038034439087, -0.2581075131893158, 1.1484850645065308, -1.2290107011795044, -0.1905147135257721, 1.6455159187316895, 2.526390790939331, 0.7409794926643372, 0.28307071328163147, 0.7293770909309387, 1.776404857635498, 1.045644760131836, 1.3887970447540283, -0.060874566435813904, -0.08271007239818573, -0.36678826808929443, 1.1125925779342651, 0.16432249546051025, 1.2563568353652954], [1.1083835363388062, 0.2203259915113449, 0.1857919543981552, 1.479049801826477, 0.7643568515777588, -0.5849335193634033, 0.32303640246391296, -1.3183737993240356, -0.9617506861686707, -0.28448379039764404, -0.23300744593143463, 0.6246893405914307, 2.3037827014923096, 1.2339091300964355, 0.16620919108390808, -0.9470763802528381, -0.36830779910087585, -0.41338953375816345, -0.43920764327049255, 0.10662342607975006, 0.3244282901287079, -1.1308268308639526, -1.2865383625030518, -2.2687582969665527, 0.21463897824287415, 0.5757613182067871, -0.3821306824684143, -1.828718662261963, 1.5959184169769287, -0.7393389940261841, -1.8726513385772705, -2.4063639640808105, -0.5737349987030029, -0.5091580152511597, 0.7273969054222107, -1.4510021209716797, -1.1448434591293335, 0.2836422026157379, -1.0546547174453735, -0.04529661685228348, 1.207783818244934, 0.7516273260116577, 0.5698409080505371, -0.3642899990081787, -0.8157400488853455, 2.0730111598968506, 0.9662895798683167, 0.10067318379878998, -0.6041083931922913, 1.043483853340149, -0.3943984806537628, -0.10562428086996078, -0.37506788969039917, 0.7711583971977234, 0.03505605831742287, 0.03729821741580963, 0.28531548380851746, 0.04382984712719917, 1.0298820734024048, 0.4934878647327423, -0.23581376671791077, 0.9909600615501404, 1.7649818658828735, -0.033627331256866455], [-0.5366090536117554, 0.5455461740493774, 0.829662024974823, 0.15502667427062988, 1.2798349857330322, 0.17880061268806458, -1.0129640102386475, 0.7665188312530518, -0.13633207976818085, 1.4104403257369995, -0.6222991347312927, -0.33303797245025635, 0.4569052755832672, -0.24586904048919678, 1.4604347944259644, -1.6665018796920776, 0.40082478523254395, -0.11574430763721466, -0.3964347839355469, -0.5438617467880249, -0.7001386284828186, -1.5770901441574097, -0.4721076190471649, -1.6324137449264526, -0.813937246799469, -0.380790650844574, 1.13435959815979, 0.5894919037818909, 1.1949125528335571, -0.5080609321594238, 0.862160325050354, 0.5768901705741882, 0.5057162642478943, -0.8592658042907715, 0.8391014933586121, 0.04063224047422409, -0.2239207625389099, 0.21602821350097656, -0.5974836945533752, 0.0488007627427578, -2.323639392852783, 0.7812941074371338, -0.4226010739803314, 0.16889958083629608, 0.1437331736087799, -0.7660278081893921, 0.2527029812335968, 0.4535689353942871, 0.9014123678207397, -2.3315327167510986, 0.31779468059539795, 0.8265218138694763, 0.3733198046684265, -0.8758695125579834, 0.12887714803218842, -0.8950531482696533, -2.172699213027954, -0.06543075293302536, -1.5641037225723267, 0.24730046093463898, -2.9579851627349854, 0.5828346014022827, -0.5362000465393066, 0.6107867360115051], [0.681577742099762, 0.17916299402713776, 1.9461195468902588, -0.492319256067276, 2.8040707111358643, -0.9666206240653992, 1.554063081741333, -1.699135422706604, 2.467026472091675, 0.4696972966194153, 0.21688027679920197, 0.19747088849544525, -0.2391968071460724, -1.2565088272094727, 1.5086743831634521, 0.4009767472743988, -0.31946444511413574, -1.519670844078064, -0.3989097476005554, 1.5836329460144043, 0.31568241119384766, -1.0674810409545898, -0.7202355861663818, -1.1751607656478882, 1.1331546306610107, 0.6194993257522583, 1.2527493238449097, -1.2968498468399048, 1.2801276445388794, -0.054983898997306824, -2.3262670040130615, 0.3141885995864868, 2.3551251888275146, -0.11898291856050491, -0.3036603629589081, -0.9838532209396362, -0.3316502869129181, 0.6720881462097168, -0.767992377281189, -0.0052634612657129765, 0.05174781009554863, -0.16876830160617828, 0.8538529276847839, 0.8062400221824646, 0.07329647988080978, -0.14867380261421204, 0.8683633208274841, -0.5450869202613831, -0.6845923066139221, -2.4584991931915283, 1.170720100402832, 1.8031365871429443, 0.03524458780884743, -0.6873987913131714, -0.11644431203603745, 0.09441262483596802, 0.8943155407905579, 1.136289119720459, -1.0802838802337646, -0.1669129580259323, 0.46682047843933105, 0.5242931842803955, 1.5542211532592773, -0.3382958173751831], [1.9414875507354736, 0.25548484921455383, -0.744166910648346, -0.5623660683631897, 0.2321438491344452, 2.253807783126831, -1.656154990196228, -0.5361186265945435, -1.7705636024475098, 0.4905086159706116, -0.07766669988632202, 1.5194705724716187, 1.45702064037323, -0.26982763409614563, -0.36737120151519775, -0.7770640850067139, 1.6187578439712524, 0.539843738079071, 0.3788087069988251, -0.7035350203514099, -0.07116708159446716, 1.3685885667800903, 0.030846308916807175, -0.78996342420578, -0.43544718623161316, 0.9654257893562317, -0.3514518737792969, -0.6257038116455078, 0.07296337932348251, 0.18347392976284027, -1.746363878250122, -1.1073169708251953, 0.800602376461029, 0.6641515493392944, -1.3742148876190186, 0.3964708745479584, -1.2509232759475708, 1.413158655166626, -1.0975186824798584, -1.2289707660675049, 0.552068293094635, -0.44604888558387756, 2.340092897415161, 0.05582983419299126, 1.197072982788086, -0.2728012502193451, 1.2642590999603271, -0.8009305000305176, 0.5403043627738953, 0.6569872498512268, -1.0527137517929077, 1.4893139600753784, -0.9098077416419983, 0.027910083532333374, -0.9614708423614502, -1.2308270931243896, -0.2576032280921936, 0.7732706665992737, 0.1153232604265213, 0.9721613526344299, -0.46170544624328613, -0.5301942229270935, 0.03278322517871857, 0.6066279411315918], [-0.645334780216217, 2.064819574356079, -1.126423954963684, 1.5227289199829102, -0.5660292506217957, 0.06748689711093903, 0.8975372910499573, 0.025996578857302666, -0.15911424160003662, -0.45906639099121094, 1.2588163614273071, -2.2097039222717285, 0.9083824753761292, -0.978872537612915, 1.127191424369812, -0.8536510467529297, 1.01261568069458, 0.3675178289413452, 0.10634993761777878, 2.1391868591308594, -0.592734694480896, 1.6223875284194946, 0.019992131739854813, 0.7089466452598572, 0.3426622748374939, -0.07905963063240051, -2.808067798614502, 0.24305884540081024, 0.32256263494491577, -0.5860868096351624, -1.358487606048584, -1.3082878589630127, 0.9997290968894958, 0.9781519770622253, -0.3233678638935089, 1.3274271488189697, -0.2985273599624634, -0.9038345217704773, -0.52109694480896, -0.279598593711853, 0.2981403172016144, 1.0763037204742432, -0.9012172818183899, 1.8657997846603394, -1.324616551399231, -1.345104694366455, 0.8847923874855042, 0.1957685947418213, 1.337180733680725, 0.38848283886909485, 1.8252500295639038, 0.7694784998893738, -1.8068386316299438, 0.3406088352203369, -0.14530859887599945, 0.2729716897010803, 0.7982681393623352, 0.1677040308713913, 0.22491870820522308, 0.3791939318180084, 0.28289321064949036, -0.165657639503479, 0.6793205142021179, -0.0008086949237622321], [-0.04506073147058487, 0.31042933464050293, -0.43369466066360474, 0.5486137866973877, 1.1322909593582153, -0.5282222032546997, -0.52252197265625, -0.9111389517784119, -1.479047179222107, -0.6212582588195801, 0.5732083320617676, 1.3363090753555298, -0.20450004935264587, 1.0535880327224731, 1.4314219951629639, -0.4836886525154114, 1.1660311222076416, 1.175645351409912, -0.8618245720863342, 1.9849637746810913, -0.26663118600845337, -0.7952647805213928, 0.39239752292633057, 0.17753131687641144, 0.9175973534584045, -0.7496654987335205, 0.09187708050012589, 0.069730244576931, -0.24998041987419128, 1.1970075368881226, 1.1917643547058105, 0.3675290048122406, 0.03182706981897354, -0.4021257162094116, 1.3083875179290771, -0.7593428492546082, -0.3208886682987213, -0.6835439205169678, 1.0131065845489502, 0.3766389787197113, 0.06191127374768257, 0.7383254766464233, 0.7510684132575989, 0.4288472533226013, -0.4894750714302063, 1.371343970298767, -0.17592814564704895, -1.9117993116378784, 0.9518190026283264, -1.2578483819961548, -1.2559908628463745, 1.0356026887893677, 0.16926859319210052, 0.42226889729499817, -0.40011855959892273, -0.2920112609863281, 0.3165417015552521, -1.1975950002670288, 0.699226975440979, 0.0882965698838234, -0.052409179508686066, 0.36133477091789246, -0.038475509732961655, 0.6271776556968689], [0.3512386381626129, -1.0004931688308716, -0.9853667616844177, -1.218263864517212, 1.509892225265503, 1.1754341125488281, 0.6010985374450684, 0.8469715714454651, 0.026067111641168594, 0.06234583631157875, 0.06663759052753448, -0.15526174008846283, 0.4137287735939026, -0.6719217300415039, 1.4494651556015015, 0.4057605266571045, 1.6204681396484375, -0.018848348408937454, -2.2353711128234863, 1.5627714395523071, -0.5563486218452454, 0.2807866930961609, -1.5885590314865112, 2.551952838897705, 1.4152461290359497, 2.607841968536377, 2.146479368209839, 1.4193696975708008, 0.4742231070995331, -1.819532036781311, -1.9494528770446777, -0.8654971718788147, 0.7541664838790894, 1.1712878942489624, 1.0573559999465942, 0.3868563771247864, 0.9541714787483215, 0.15196266770362854, 2.648982286453247, -0.4762451946735382, 0.5713325142860413, 0.6690197587013245, 0.3158003091812134, 1.038532018661499, -1.6902135610580444, -0.4084644913673401, -1.1216896772384644, -0.8361480236053467, 0.38289541006088257, 1.3284858465194702, 1.1499987840652466, 0.29187875986099243, -1.1958961486816406, 0.2879379093647003, 0.38313066959381104, -0.1319497972726822, -0.1258477419614792, 1.3269319534301758, -1.0252286195755005, 0.7121545672416687, 0.30050936341285706, 0.22624574601650238, -0.6536497473716736, -0.35750770568847656], [-1.4127832651138306, 1.1176114082336426, 0.07650915533304214, 0.4733705520629883, -0.06331437081098557, 1.2742160558700562, -0.10936406254768372, -3.391413450241089, -0.25759434700012207, 1.3536498546600342, 0.08470389991998672, 1.1632972955703735, -0.9635977745056152, 0.8160532116889954, -0.5936017632484436, -1.2993693351745605, 0.2153460681438446, 0.0010689834598451853, -0.988018810749054, -0.669884979724884, -0.3410414159297943, -0.555499792098999, -0.25213760137557983, 0.5657253861427307, 0.8377913236618042, -1.0337966680526733, 0.684309720993042, -0.17229028046131134, 0.6707252860069275, 1.0754421949386597, 0.30650922656059265, -0.8414115309715271, 1.4821873903274536, 0.2549263536930084, -0.10171347111463547, 1.1038013696670532, -0.1977960169315338, -0.12130933254957199, 0.6889570355415344, 0.9484891891479492, 1.4438775777816772, -1.876318097114563, -0.2703145146369934, 1.4430197477340698, 0.27195674180984497, -0.05749206244945526, -1.2318757772445679, 0.7323459982872009, -0.12851153314113617, -0.32956966757774353, 0.21458229422569275, 0.011427387595176697, -0.4932209551334381, 0.9831504821777344, 0.19111137092113495, 0.1737644225358963, -0.30287671089172363, 0.28022363781929016, 1.9836807250976562, -0.8092536926269531, -0.022195497527718544, 2.2513723373413086, -0.9359358549118042, -0.3337382674217224], [-0.598095178604126, 0.6024354100227356, -0.2773284912109375, -1.8899520635604858, -0.08216604590415955, -0.029348645359277725, -1.9540103673934937, -0.6239851713180542, -1.817116141319275, -0.893903374671936, -0.6373786926269531, 1.6719671487808228, -0.12035417556762695, -1.053553581237793, -0.7134082317352295, 0.44461485743522644, 1.0311880111694336, -0.014280674047768116, 0.05948713794350624, 0.5082426071166992, -0.3964696526527405, -0.03182728588581085, -1.2765733003616333, -0.5128871202468872, -0.9357249140739441, 1.2482088804244995, 0.39395004510879517, 1.415439248085022, 1.0699818134307861, 0.32208284735679626, -2.166750192642212, 1.191498041152954, 2.483323574066162, -0.14160576462745667, 1.5739909410476685, 1.1576073169708252, 1.1723066568374634, -1.609985113143921, 2.0161638259887695, 1.157447099685669, 0.630375862121582, 0.39086538553237915, 3.1927402019500732, -0.5812934041023254, 1.349752426147461, 0.0026829889975488186, -1.0109999179840088, 1.3134499788284302, 0.7174093127250671, 3.3254573345184326, 1.3913774490356445, -1.102957010269165, 0.3155041038990021, -0.04812176525592804, 1.601879596710205, 1.3438246250152588, -0.9418633580207825, -0.1310996562242508, 0.9286766052246094, -0.26848816871643066, -1.6850076913833618, 0.5145537853240967, -0.3117940127849579, -0.6221182942390442], [-0.24400827288627625, -0.0901261568069458, -0.942894458770752, -0.8056818842887878, -0.8558711409568787, 0.87247633934021, 0.6521696448326111, -1.7765330076217651, -0.3767804801464081, 2.131225347518921, -0.07206222414970398, -0.9148926734924316, 2.066781997680664, 0.551571249961853, 0.45691362023353577, 0.21945860981941223, 0.5581702589988708, -0.8910560011863708, -0.9588321447372437, -0.1633712202310562, 0.3758206069469452, -1.227475643157959, -0.3943638205528259, 0.7789929509162903, 0.7161015868186951, 0.6213851571083069, -0.9493169784545898, -0.01774708554148674, 1.40794837474823, 1.7239688634872437, -0.6657387018203735, 0.5485787987709045, -0.551937460899353, 0.13510455191135406, -1.2571686506271362, 0.3697699308395386, -0.28399327397346497, -0.7367295026779175, -0.35784676671028137, -0.2642868161201477, -0.3357912600040436, 1.5343446731567383, 2.1460020542144775, 0.6283589005470276, -0.03053675964474678, 0.5476117134094238, -0.06527871638536453, 1.2052652835845947, 1.9879788160324097, -2.2365522384643555, -0.3856629729270935, 0.21004115045070648, -0.5572982430458069, -0.5428950786590576, 1.1682265996932983, -1.316503882408142, 0.5298817157745361, 1.649560570716858, -0.08768755942583084, -1.130967617034912, -0.4056423604488373, -1.0634459257125854, 0.4963698983192444, -1.2568819522857666], [-0.5354689359664917, 1.3565831184387207, -0.48235276341438293, -0.7553310394287109, 1.2668730020523071, -0.4243859350681305, -1.9425439834594727, 0.4855886995792389, 0.27650681138038635, -0.9270082116127014, -1.2054815292358398, -0.18888933956623077, 0.5741717219352722, -1.5175504684448242, -0.6088846325874329, -0.021183060482144356, -0.13398543000221252, -0.7225058674812317, -0.3143637776374817, -0.3890942931175232, 1.0018737316131592, -1.5433958768844604, -0.9131922721862793, -0.7563002705574036, -0.48646533489227295, -0.5977649688720703, -0.9153134226799011, -1.5118670463562012, 0.44736114144325256, -0.2724616229534149, 2.628309726715088, 0.6508125066757202, -0.5332870483398438, -0.23977547883987427, 1.638375163078308, 1.080083966255188, 0.661141037940979, -1.1242204904556274, -1.7739204168319702, 0.1872708946466446, 0.15825963020324707, 1.3124550580978394, -0.2387063056230545, 0.9677916765213013, 1.293826937675476, -0.06110500916838646, 0.9056085348129272, 0.5257302522659302, 1.262699007987976, 0.39442768692970276, 0.9063008427619934, -0.9589407444000244, -1.147925853729248, 3.2184925079345703, -0.6781047582626343, 0.9259183406829834, 0.671582818031311, -0.26991328597068787, 0.5085119605064392, -1.061203122138977, -0.13068024814128876, -0.34250855445861816, -0.34828656911849976, -0.6729540228843689], [0.9476821422576904, -0.6623585820198059, -1.531180739402771, 1.073649525642395, -0.793123185634613, 1.3723855018615723, -0.46931177377700806, -0.25685927271842957, 0.45704707503318787, -0.18514922261238098, 0.5328397154808044, 0.2722175419330597, 0.596049964427948, -0.07897572219371796, 0.8824439644813538, 0.15817104279994965, -0.5470131635665894, 0.04379059374332428, -1.0484323501586914, 0.6492214798927307, -1.230393409729004, -0.9249172210693359, -0.17896750569343567, 0.09416987746953964, -1.131381630897522, -0.7343423366546631, -0.4099670648574829, -0.17892992496490479, 1.1425976753234863, 0.4081166386604309, 1.3528900146484375, -1.0294606685638428, 0.9607509970664978, 1.7693418264389038, -0.4821373224258423, 0.7668755054473877, -0.011815352365374565, 0.08907979726791382, -1.2395321130752563, 1.7045483589172363, 3.9065754413604736, -1.1826125383377075, 1.942348837852478, -0.03002188913524151, 0.17704123258590698, 1.0177544355392456, 0.45740920305252075, -2.4734339714050293, -0.17718087136745453, -0.10813194513320923, 0.5190110802650452, 0.11089633405208588, -0.6208486557006836, 0.9243350625038147, 1.9853051900863647, 1.233386754989624, -0.8742731809616089, -0.6309731602668762, 1.0690340995788574, 1.0449048280715942, -0.09740573912858963, -1.6521327495574951, -1.2230794429779053, 0.21724797785282135], [1.2472602128982544, 2.0816171169281006, 0.8973526954650879, 0.14144451916217804, -0.4433651268482208, 0.2706664800643921, -0.9947795271873474, 1.2253907918930054, 0.25630342960357666, -0.6121773719787598, -0.04195169359445572, -0.45963743329048157, -0.055883582681417465, -0.9300518035888672, -0.604354739189148, 0.25367236137390137, 0.9481699466705322, -1.4876292943954468, -1.1993954181671143, 1.5746021270751953, -2.3315892219543457, -0.8390324115753174, -0.34926968812942505, 1.019949197769165, -0.4738216996192932, -1.3744804859161377, 0.18863534927368164, 0.04918856918811798, 1.3796334266662598, 0.2072126269340515, -1.9084548950195312, 0.13086122274398804, -0.7317343950271606, 0.2207670956850052, 1.9959830045700073, -0.16182543337345123, 1.1193645000457764, 2.5712459087371826, 1.5416419506072998, -0.5456997752189636, 0.884666919708252, 1.049162745475769, -0.8175708055496216, 2.143378734588623, -0.741100549697876, 1.5051031112670898, 0.15699340403079987, -0.43746790289878845, 0.6348415613174438, 0.13645413517951965, -0.12805752456188202, 1.2574149370193481, -1.5235400199890137, 0.0864538624882698, 0.16085787117481232, -0.9444403648376465, 0.24291974306106567, -1.3055981397628784, -0.40796133875846863, 0.5880256295204163, 0.8253968954086304, 1.6957510709762573, 0.4777573049068451, -0.4864788353443146], [2.1529974937438965, -1.6062999963760376, -0.15821295976638794, 0.7185133695602417, 1.514072060585022, -0.6569004058837891, 0.9010493159294128, 1.202795386314392, 0.20830672979354858, 0.6556782722473145, -0.9068578481674194, 0.9551786780357361, 1.477643370628357, 0.061827052384614944, -0.48854443430900574, 0.9269132614135742, -0.6045361161231995, 1.4417433738708496, 1.7318058013916016, 1.3091918230056763, -0.24354423582553864, 0.08812735229730606, 1.2969295978546143, 0.7483561038970947, -0.6021938323974609, -0.08194874227046967, 0.8198509216308594, -0.568316638469696, -1.1152831315994263, 1.9773505926132202, 0.0623023547232151, -0.2321542650461197, 0.5930243134498596, 0.7772778868675232, -0.19485390186309814, -0.15914377570152283, 0.46888744831085205, -0.4254147708415985, -0.7909963130950928, -1.7575368881225586, 2.035315752029419, -1.0432860851287842, 1.0052169561386108, 0.8470394611358643, 0.8779729604721069, -0.8555954098701477, 0.03000662662088871, 0.2920163571834564, 0.2997063994407654, -0.6921019554138184, 0.29088497161865234, 0.6404786705970764, -1.1216812133789062, 0.6081863045692444, -2.4311130046844482, -0.1275518834590912, -2.207341432571411, 1.2645010948181152, -0.488877534866333, 0.7496397495269775, -0.8185490369796753, -0.18734316527843475, -0.9976105690002441, 1.3820124864578247], [1.8094909191131592, -0.05060417577624321, 0.5880098938941956, 0.9652719497680664, 0.9046555161476135, 0.8484171628952026, 0.32836902141571045, -1.5186944007873535, -0.5530908107757568, 0.14564669132232666, 0.716874361038208, -0.8963783979415894, -1.858721375465393, 2.72309947013855, -0.9530938863754272, -0.21159657835960388, 0.48056358098983765, 0.8059362769126892, -1.2299082279205322, -0.6693846583366394, -0.022534113377332687, -0.2709383964538574, -0.33614298701286316, -0.10924313962459564, -0.4740164279937744, -0.9465204477310181, -1.4539324045181274, 0.7733983397483826, -0.9500275254249573, -0.1991579234600067, -0.37124335765838623, -0.6938588619232178, 0.02657511830329895, -1.207729458808899, 0.05748631805181503, -0.997146487236023, -1.0219398736953735, 0.03544261306524277, -0.2652716338634491, 0.3643403649330139, -1.304310917854309, 0.23415684700012207, 0.5981485247612, -0.35261908173561096, -0.21341143548488617, -0.6544402837753296, 0.17833945155143738, 0.8336470723152161, 1.3431707620620728, -1.5401878356933594, 0.47040989995002747, 0.39270827174186707, 1.9995583295822144, -0.0807952806353569, 0.4480860233306885, -0.2657034397125244, 2.1409592628479004, -1.235547661781311, 0.9642096757888794, -0.018671827390789986, 0.3656664490699768, -0.15586699545383453, 0.4805585443973541, -0.9210078120231628], [-2.6278507709503174, 2.5496931076049805, 0.005935055203735828, -0.46373701095581055, -1.2975746393203735, -1.0367553234100342, -2.21559739112854, 1.3496366739273071, -0.44704365730285645, 0.12789587676525116, -0.44821277260780334, -1.0628329515457153, -0.42091843485832214, -0.8685920238494873, 0.3844996988773346, 0.05570201948285103, -0.042759090662002563, -0.7670208811759949, 0.4943861663341522, 0.6799913048744202, 0.9982398152351379, -0.9947280287742615, 0.9383995532989502, -1.6566253900527954, 0.5929868817329407, -1.2797598838806152, -1.0210336446762085, -2.213294744491577, 1.0516008138656616, -0.9030901193618774, -2.2484397888183594, 1.0430874824523926, 0.5283793807029724, 0.8385611176490784, -1.023903727531433, -0.5903439521789551, 0.6012613773345947, -0.5898663401603699, 0.9568302035331726, -0.43945276737213135, -0.6045883297920227, -0.5006972551345825, 0.8979273438453674, -0.8094447255134583, 1.4345810413360596, 0.651024580001831, 1.7633146047592163, -0.9504519701004028, -0.271171897649765, -0.9973727464675903, 0.2586846947669983, -0.5017490983009338, 0.35931694507598877, 0.7445248961448669, -0.08846363425254822, 1.9287692308425903, -0.1230306401848793, 1.0293011665344238, -0.7758191823959351, 0.12878617644309998, 0.9619888067245483, -1.8601791858673096, 1.7823153734207153, 0.6560080051422119], [0.4295738935470581, 0.7162919640541077, -1.8282090425491333, -0.018564896658062935, -0.11892376095056534, -0.9817420840263367, -0.33720147609710693, -0.5584213733673096, -0.0015810664044693112, -2.4508378505706787, -1.0293643474578857, 0.013540658168494701, 1.4047222137451172, 0.15156672894954681, 1.0332441329956055, 0.12009529769420624, -0.43112871050834656, -0.6737098097801208, -0.0282637607306242, 1.8195006847381592, 1.3994884490966797, 0.4392361342906952, -2.104304552078247, 1.686773657798767, 0.6858843564987183, 0.40704506635665894, 0.677914559841156, -0.9132474064826965, 0.004760894924402237, 0.3288895785808563, -1.1862767934799194, -0.34655413031578064, -0.20636717975139618, 1.7676461935043335, -0.4712144732475281, -0.6669069528579712, 0.9768766760826111, 0.25985148549079895, 0.9367820620536804, -0.4866001009941101, 0.07031948119401932, 0.08565045893192291, -0.7496758103370667, -0.29451796412467957, -0.2809397280216217, -2.7188186645507812, -3.421048879623413, 0.4439442455768585, 0.674993634223938, -0.5361945033073425, -0.16048744320869446, 0.43514639139175415, 1.852588415145874, -0.18544436991214752, 0.3894141614437103, -1.0920865535736084, 1.1370439529418945, 0.5557866096496582, -0.9062309265136719, 0.9156282544136047, -1.192042589187622, -0.5352351665496826, 0.8119566440582275, 1.3822431564331055], [-0.06737996637821198, -0.4403434097766876, 0.10915114730596542, -0.2172154188156128, -0.027140866965055466, 0.5039420127868652, 0.9700084924697876, 0.4816296696662903, 1.1572386026382446, -0.9102766513824463, 1.525375485420227, 0.9847257733345032, -0.25427210330963135, -0.36865806579589844, -0.7463054060935974, -0.09784317761659622, -1.6690208911895752, 0.22746005654335022, -1.4089058637619019, -0.32843634486198425, -1.261568307876587, -0.39688587188720703, 0.028108661994338036, -0.6228126287460327, -0.2898768484592438, 1.5997800827026367, -0.2628301680088043, 0.9779992699623108, 0.5280701518058777, 0.2933874726295471, 0.45137473940849304, -0.26215648651123047, -0.7096350789070129, 0.29595473408699036, -0.16650596261024475, -0.2913067042827606, 0.7257257103919983, -0.07702738046646118, -0.12384269386529922, 2.0671379566192627, 1.7681281566619873, 1.498212218284607, -0.6419981122016907, 1.1325465440750122, -1.4571614265441895, -0.9099966287612915, 0.5277962684631348, -2.188598394393921, -0.1575174480676651, -0.34353312849998474, -1.138153076171875, 1.0580997467041016, -0.10086781531572342, 0.4884122908115387, -1.38637113571167, -1.2671700716018677, 1.4335963726043701, 0.3853890895843506, -0.9100576639175415, 1.3517895936965942, 0.729476273059845, -1.0752042531967163, 0.4649510681629181, -0.12694254517555237], [-0.8419642448425293, -0.912154495716095, -0.7675715684890747, 0.05462171137332916, -1.382104754447937, -0.8690273761749268, 0.6443690657615662, 0.8422063589096069, -1.8963111639022827, 0.44742730259895325, -0.6244248151779175, -0.9037641882896423, 1.1501566171646118, 0.3661643862724304, 0.1382589340209961, -0.002600703854113817, -0.3907822072505951, 0.13164126873016357, 0.5640061497688293, -1.1916764974594116, 0.682824432849884, 1.124679684638977, -0.03569502755999565, 0.22974437475204468, 1.1322022676467896, 0.2907909154891968, 1.0476741790771484, -0.6562737822532654, -1.840775489807129, 0.43835294246673584, 0.9778735041618347, -0.2936423420906067, 2.352020263671875, -0.44736406207084656, -2.4943902492523193, 0.38711321353912354, 0.7234074473381042, -1.9536792039871216, -0.6106313467025757, -0.17022567987442017, 0.08744502067565918, -1.776410460472107, 1.718460202217102, 0.6810427308082581, -1.0212262868881226, -0.6921772956848145, -0.6307689547538757, -0.0423738993704319, 0.5559182167053223, -0.5834449529647827, 0.6743927597999573, -1.081047773361206, -0.802614688873291, 0.06866458803415298, 0.07349956780672073, 0.04797784611582756, -1.2359145879745483, 0.6326936483383179, 0.8560073375701904, 2.0166423320770264, 0.061766140162944794, -0.4230457544326782, -0.28388547897338867, -1.5765869617462158], [2.2428946495056152, -0.5253207087516785, -1.341378927230835, -1.9341766834259033, -2.6716718673706055, 1.9502017498016357, -1.3584996461868286, -0.2434462308883667, -1.3422154188156128, 1.4300111532211304, -1.4742289781570435, 0.844006359577179, -1.0368187427520752, 1.0053939819335938, 0.37184661626815796, 0.12528011202812195, 1.1237584352493286, -0.3994734287261963, 1.2398649454116821, -0.4193485379219055, 1.8191571235656738, 1.558246374130249, -0.13446356356143951, -1.0733145475387573, 0.5053298473358154, 0.6878432035446167, 0.013919062912464142, 0.12384574115276337, 1.488326072692871, 0.1486697942018509, -0.5349751114845276, 1.0694926977157593, 0.6960859298706055, 0.7055765986442566, -0.2861555218696594, -0.23481601476669312, -0.3100275695323944, 0.6480212807655334, 0.5929811000823975, 0.008750427514314651, 0.4436439871788025, -1.6057554483413696, 1.4489085674285889, -0.18612627685070038, -1.4989100694656372, -0.5269847512245178, 0.1971871256828308, -0.18623748421669006, -0.3961544930934906, -0.7581121921539307, 0.9134637713432312, -0.7630305886268616, -0.3757021427154541, 0.5530399680137634, 0.43473145365715027, 1.738358497619629, 0.2052128165960312, 0.015549831092357635, -0.13742467761039734, -0.3726601004600525, 1.0012820959091187, -1.1745201349258423, 1.4581841230392456, -0.1157948449254036], [-0.4373285472393036, 1.174480676651001, -1.6681195497512817, -2.259716033935547, 0.441644549369812, -0.5453998446464539, -0.718796968460083, -0.7071208953857422, 2.5965466499328613, -0.3408818244934082, 0.22732491791248322, -0.47155770659446716, 0.9817019104957581, -2.254185914993286, 0.01026537548750639, 0.15870709717273712, 0.16378958523273468, -0.6817684769630432, -0.5911071300506592, -1.3413610458374023, 0.35489514470100403, 0.7071198225021362, 0.563640296459198, 0.4194454252719879, 0.8077433109283447, -0.5644192695617676, -2.0261242389678955, -1.1559553146362305, 0.6748518943786621, -1.614227056503296, -0.33037954568862915, 0.7956116795539856, 0.4691131114959717, 0.3756594955921173, 0.12572580575942993, -0.19879822432994843, -0.5436007976531982, 1.0954660177230835, -0.6475323438644409, -0.38374385237693787, 0.3564468026161194, -0.9870404005050659, 1.0959246158599854, -0.7061366438865662, 0.5142384767532349, 0.4649865925312042, 0.5511168837547302, -1.962419867515564, -0.1752261370420456, -1.3275213241577148, 1.1064823865890503, -0.3991255462169647, -0.7628606557846069, -0.002629605820402503, -1.3137538433074951, -0.5771215558052063, 0.802432656288147, -0.08431243151426315, -0.3029020428657532, -0.34957951307296753, -1.3556073904037476, 0.1275235116481781, 0.2169194370508194, 0.48584792017936707], [1.5716564655303955, 0.4374292194843292, -0.3085983991622925, -0.9349175691604614, 1.221696376800537, -1.460432529449463, -1.8829360008239746, 0.04460615664720535, 0.6802930235862732, -0.17795397341251373, 0.4738124907016754, 0.8559793829917908, 0.2493342012166977, 0.557230532169342, 0.339269757270813, 0.22931674122810364, 0.5362313985824585, 0.41445934772491455, 1.0067310333251953, 0.15056589245796204, 0.2760660648345947, -0.1620364636182785, 0.3553749620914459, 0.2402111440896988, 0.07863610237836838, -0.3969539403915405, -0.4771097004413605, 0.28351113200187683, -1.1647883653640747, 1.498373031616211, -1.0288753509521484, -0.30239760875701904, -0.2524484694004059, -1.2141673564910889, 1.9489558935165405, 0.3933255970478058, 0.20358823239803314, -0.22180145978927612, -0.28870636224746704, 0.2958439588546753, 0.1281808316707611, 1.7494152784347534, -0.3089979588985443, 1.214882731437683, -0.9814590215682983, -0.3422603905200958, -2.1367740631103516, -0.6467220783233643, 0.5106819868087769, -0.05194223299622536, -0.8428699374198914, -0.06244993209838867, 0.0033359676599502563, 0.737392008304596, 0.4398391842842102, 1.1883697509765625, -0.49714991450309753, -2.155665636062622, 0.48407790064811707, -0.542877197265625, 0.6977649927139282, 0.22334259748458862, 2.156930446624756, -0.9470216035842896], [0.5213946104049683, 0.878435492515564, -0.7412397861480713, -0.43344688415527344, 0.6397995948791504, -0.9268394708633423, -0.5323318243026733, -0.28972509503364563, -0.48805326223373413, 2.128096580505371, -0.44771671295166016, -0.17407451570034027, 1.3045259714126587, -1.3710154294967651, 1.0378222465515137, 1.1037472486495972, -0.5205230116844177, -0.882476270198822, 1.4785679578781128, -1.1458251476287842, 1.3135823011398315, -0.7311722040176392, 0.6484338641166687, 0.49036848545074463, -0.4711792469024658, -0.7880083918571472, 0.7780317664146423, 0.4536241888999939, 0.1668097972869873, 0.4313453435897827, 0.8795941472053528, -0.39947402477264404, 0.6127195358276367, -0.29536065459251404, -0.7783926129341125, 0.845238208770752, 0.19942353665828705, 1.6123219728469849, -2.8463690280914307, -2.00535249710083, -0.686555802822113, -0.4530315101146698, -0.04518552124500275, 0.6790320873260498, -0.1354692429304123, -1.3950016498565674, -1.5138219594955444, -0.0865895226597786, -0.044502727687358856, -1.0055469274520874, -0.8830639719963074, 1.3036445379257202, 1.2979971170425415, -0.780258297920227, -0.06529895216226578, -1.2160258293151855, 0.9804141521453857, 2.6558706760406494, -0.7529322504997253, 0.9693494439125061, -0.13929729163646698, -1.7261326313018799, -2.3385345935821533, 0.8997415900230408], [1.8707311153411865, 0.19717448949813843, -0.62501460313797, 0.872579038143158, -0.4710133969783783, -1.8484591245651245, 1.327553391456604, 1.4425112009048462, -0.39662736654281616, 0.9135845303535461, -1.344123363494873, -0.8965219855308533, 0.20494972169399261, -1.1546127796173096, -2.08966326713562, -0.854852557182312, -0.4109383821487427, 0.16618837416172028, -0.5670083165168762, -0.8748676180839539, -0.6056921482086182, -0.6751745939254761, 0.10732252895832062, -1.8899662494659424, 0.9094662070274353, 0.4487168490886688, -1.6866719722747803, -0.20587025582790375, 0.40071508288383484, 0.45063456892967224, -0.2581254243850708, 1.4587277173995972, -0.5468409061431885, -0.4933532774448395, -0.6569746136665344, -1.5347365140914917, 0.4998664855957031, 0.6615411639213562, -0.9343660473823547, -0.0939956083893776, 0.00820013228803873, -0.21018528938293457, -2.0609633922576904, 1.2049212455749512, -0.8054609894752502, 0.5878974199295044, 0.11364945769309998, -1.6185606718063354, -0.8246118426322937, 2.094622850418091, -0.5373024940490723, 1.5218284130096436, -1.3227087259292603, 0.36004894971847534, -0.2068040370941162, 0.10996207594871521, -0.9977946281433105, 0.09560966491699219, -0.8937236070632935, -0.42655694484710693, 0.5715469717979431, -0.5706710815429688, 1.2749145030975342, 0.1134335920214653], [0.577128529548645, -0.8084589838981628, 0.02612069807946682, 0.6580536961555481, -0.025456208735704422, 0.27413272857666016, 0.2049674391746521, 0.36215436458587646, -0.3947460949420929, -1.4581977128982544, -0.6004248857498169, 0.024496452882885933, 0.6105342507362366, -0.35016095638275146, 0.7116587162017822, -0.21071524918079376, 0.17751650512218475, 0.03575698286294937, 0.11334330588579178, 0.3886101543903351, -0.8187267780303955, -0.19921445846557617, -0.012273979373276234, 0.8466311097145081, -0.7642927765846252, 0.6620379090309143, -1.0394105911254883, 0.07075059413909912, -1.35062575340271, 0.8056504130363464, -0.6796668171882629, -0.0005717206513509154, -1.3161624670028687, 0.5648719072341919, -1.0511475801467896, -1.2190418243408203, -0.618215799331665, -0.1221105083823204, -0.9597739577293396, -2.8020389080047607, -0.9068094491958618, 0.5816366076469421, 1.6803170442581177, 0.04028225317597389, -0.8344892859458923, 0.7099761962890625, -0.06422963738441467, -1.107647180557251, -0.3771013617515564, 1.067640781402588, -0.281171590089798, 0.03895235434174538, -0.41382095217704773, 0.05960100144147873, 0.49773019552230835, 0.3180491030216217, 0.21219556033611298, -0.7839180827140808, -1.1944040060043335, 0.25244706869125366, 0.9717355966567993, -0.21340976655483246, 0.5954524278640747, -1.309627652168274], [-0.02769818715751171, 0.8735805153846741, -0.12030457705259323, -1.671346664428711, 2.1562325954437256, 0.7813946008682251, 0.5538889169692993, -0.2653355896472931, 0.45745956897735596, 0.9896189570426941, 0.35827818512916565, 1.3975249528884888, 1.227983832359314, 0.04292307049036026, -1.2796238660812378, -0.3462028205394745, 0.017199033871293068, -0.788148045539856, -0.08110706508159637, -0.052298858761787415, 1.118095874786377, 0.0035420237109065056, -0.7818963527679443, 0.5177190899848938, 0.6433570384979248, -0.3158281445503235, 0.13782502710819244, 0.1671142578125, 1.2212101221084595, 0.6047263145446777, 0.23427186906337738, -2.514798879623413, 0.4303375780582428, -0.47909218072891235, -1.683529019355774, 1.2690597772598267, -1.1530323028564453, -0.7655835151672363, -0.21294094622135162, 0.5213974714279175, 0.6257192492485046, -0.29373008012771606, -0.7469061613082886, -1.2918297052383423, -0.965975284576416, 1.8841270208358765, -0.6269090175628662, -1.3147622346878052, -0.40020617842674255, -1.8527590036392212, 0.21676960587501526, -1.178279161453247, -0.352382093667984, 1.5057611465454102, 0.4871343672275543, -0.37886786460876465, 0.5627255439758301, -0.6688172221183777, -0.31842875480651855, -1.364943504333496, 2.9611010551452637, -1.1030066013336182, -0.6773484349250793, 0.3399587869644165], [-0.8045053482055664, 0.0830591544508934, -1.738390564918518, -1.0885076522827148, -0.2409709393978119, 0.13826227188110352, 0.37794050574302673, -0.4313625991344452, -0.07613261789083481, -0.4427393078804016, -0.9431394338607788, -0.8771867156028748, 0.4022543728351593, -0.47843822836875916, -0.6813532710075378, -0.08815360814332962, -1.5890522003173828, 0.40283164381980896, 0.46884849667549133, 0.8705736398696899, 0.5946560502052307, 0.01567268744111061, 1.7084739208221436, -0.7193080186843872, -0.1357499361038208, 1.6346863508224487, -0.6983682513237, -0.2488141506910324, 2.007208824157715, -0.40525227785110474, -1.6470102071762085, 2.2662086486816406, 0.09554456174373627, 0.47439029812812805, -0.021062450483441353, -1.4751238822937012, 0.5820277333259583, 0.35903462767601013, -1.0734401941299438, -0.9882268905639648, -0.34077373147010803, 1.5061851739883423, -0.4517654478549957, -0.8378095626831055, 1.0933536291122437, -0.7132306694984436, 0.40817686915397644, 0.22569747269153595, -1.4839973449707031, -0.05757225677371025, 1.1456005573272705, 1.5254852771759033, 2.203890562057495, 1.1367021799087524, -0.9664758443832397, -0.6074729561805725, 0.3637300729751587, -0.6913390755653381, -0.05038497969508171, -0.8473889827728271, -0.6745581030845642, 0.29162290692329407, -0.04030865058302879, -0.6629599332809448], [0.2600463926792145, 0.5858387351036072, -0.5093259215354919, -1.978222131729126, 1.4388078451156616, -0.11661873757839203, 0.3356827199459076, 0.027143599465489388, -0.5834814310073853, -0.8019217252731323, -0.8112388253211975, 0.21306821703910828, 0.3834458887577057, -0.924968421459198, 0.10887004435062408, 1.5579495429992676, 0.48931586742401123, -0.17428579926490784, 0.6493939161300659, -0.6548128724098206, 1.4794052839279175, -0.2548462450504303, -1.7622309923171997, 0.6773403286933899, -1.4627140760421753, 0.44892388582229614, 0.7695149779319763, -1.387662410736084, 0.32345178723335266, -1.5236448049545288, 0.6029654741287231, 0.8367457389831543, -0.6872469186782837, 1.2192529439926147, -0.07066536694765091, 0.2641107141971588, 0.18698087334632874, 0.7840594053268433, 0.5463611483573914, -1.2148514986038208, -0.30071717500686646, -0.9163714051246643, -1.8444271087646484, -0.20011310279369354, 0.14766713976860046, 1.2226020097732544, -0.48284417390823364, -0.6474372744560242, -0.1302398294210434, 1.0071394443511963, -1.8600624799728394, 1.8693686723709106, 0.6038024425506592, 0.02398390881717205, 1.1552976369857788, 0.6913525462150574, 0.6158667206764221, 0.5876365303993225, -1.5378758907318115, 1.3801625967025757, -1.044963002204895, -0.3282926082611084, -0.27601906657218933, 1.4495877027511597], [-0.7175120115280151, -0.8443800210952759, -0.48353880643844604, 0.33854377269744873, -0.5638027191162109, 1.238111972808838, -1.630008578300476, -0.8205034732818604, 0.09142009913921356, -1.6699693202972412, -0.0887613520026207, 0.1463635116815567, 1.444821834564209, 0.35153913497924805, -0.889127254486084, 2.3432302474975586, 1.9425355195999146, -0.12419896572828293, 1.8876205682754517, -0.6546730399131775, -0.584690511226654, -0.2192142903804779, 2.1740078926086426, -1.1561343669891357, -1.2182791233062744, 1.0996729135513306, 0.9203512668609619, 0.27072232961654663, -1.2639236450195312, -1.678247332572937, 0.2971307933330536, 1.5106970071792603, 0.5119870901107788, 1.1493581533432007, 0.23667468130588531, 0.09527434408664703, -1.004656434059143, 0.2914673984050751, -0.13959293067455292, 0.34328216314315796, -0.45443323254585266, -0.5332647562026978, 0.7217504382133484, 0.27679160237312317, 0.717190146446228, -1.267549991607666, 0.913341760635376, -0.3520072400569916, -1.0661120414733887, -0.48745518922805786, 0.4837760925292969, 1.5053712129592896, 1.5766057968139648, 1.340617299079895, -1.5624191761016846, -0.5279034972190857, -0.2676964998245239, 1.831020712852478, -1.520952582359314, -0.5103732943534851, 0.33347538113594055, 0.579261839389801, 0.47238579392433167, -0.6906167268753052], [0.5966417193412781, 1.0353761911392212, 0.07109416276216507, -1.062543272972107, -1.2649701833724976, -1.1550359725952148, 0.09987052530050278, 0.1569690853357315, -1.0756944417953491, -0.4033085107803345, -1.8451238870620728, -2.1604013442993164, 0.524070680141449, -0.8696748614311218, 0.08679481595754623, -1.5080221891403198, -0.702509343624115, 0.5839076638221741, -0.93473881483078, 0.7975203990936279, 0.6398343443870544, -0.19408144056797028, 1.679792046546936, -0.07443608343601227, 0.49682217836380005, -0.392447829246521, -0.21092885732650757, 0.34333112835884094, -0.4183201491832733, 0.5922380685806274, -1.5037301778793335, -0.450497031211853, 1.022317886352539, 0.09936264902353287, 1.328068494796753, 0.7979673147201538, -0.29058170318603516, -2.5793285369873047, -0.3571167290210724, -0.009931010194122791, 1.6461261510849, 0.6061572432518005, -0.27662229537963867, -0.4887474477291107, 0.5495216250419617, 0.7263056635856628, -1.3555370569229126, -1.7552059888839722, -0.7946807146072388, 0.4523801803588867, -1.2860485315322876, -0.8150809407234192, 1.0974498987197876, -0.9321447610855103, 0.4543059766292572, 0.4535185396671295, 0.4417148232460022, 1.779938817024231, -2.3348047733306885, 0.2902945876121521, 1.0831732749938965, 0.696202278137207, -0.3698868453502655, -0.47625845670700073], [0.6614988446235657, -0.03932923078536987, 0.933653712272644, -0.5396538972854614, -0.2479521781206131, -0.9587856531143188, 0.31947848200798035, 1.6403042078018188, 0.5883257985115051, -0.9618178606033325, 0.12810176610946655, 0.7049450874328613, -0.3036308288574219, 0.3351612985134125, 0.7807524800300598, -0.20639903843402863, 0.8836176991462708, 0.13789629936218262, 0.2844661772251129, -0.03517985716462135, -0.49508237838745117, -0.3378554880619049, 0.866753101348877, 0.0423295758664608, 0.8212983012199402, 0.7784945964813232, 0.8479787111282349, 1.5614163875579834, 1.206215262413025, 0.10225213319063187, 0.927232563495636, -0.6115687489509583, 2.5737760066986084, -0.3202834129333496, -1.9425746202468872, -0.6281400322914124, -0.464590847492218, -0.23899823427200317, -1.1611405611038208, -0.6419064998626709, -1.3260672092437744, -0.1454027146100998, 0.4613581597805023, 1.9555104970932007, 0.9564256072044373, -0.433769166469574, 0.522836446762085, 0.5620426535606384, -1.6569080352783203, -1.5579707622528076, 0.023451559245586395, -2.1353530883789062, 2.0707318782806396, 0.3723922371864319, 0.5453136563301086, 0.05464702472090721, 0.2538047432899475, -0.13763141632080078, 0.6807628870010376, 0.2113804668188095, -0.32348179817199707, -0.7216842770576477, 0.3508789539337158, -1.046270489692688], [1.226434588432312, -1.7749583721160889, 0.32692450284957886, 0.5644770860671997, 0.10215169936418533, 0.33854177594184875, -0.9339599609375, -0.026278574019670486, 0.7574989795684814, 0.48545026779174805, 0.22832445800304413, -0.8977545499801636, -0.5618495345115662, -1.2683244943618774, 2.1077687740325928, 0.7636716961860657, 1.1615498065948486, -2.1624414920806885, -0.9844074845314026, -0.5509109497070312, 0.5290729403495789, 0.20764413475990295, -0.6315010786056519, 1.292682409286499, 1.2912795543670654, 1.3318229913711548, -0.20409415662288666, -1.4118618965148926, -0.37259143590927124, 1.2522765398025513, 1.1159712076187134, 1.7561562061309814, -1.2442251443862915, -0.009934468194842339, -1.2479552030563354, -1.2906999588012695, 0.1943080574274063, -0.12916448712348938, -0.8051021695137024, -0.056682437658309937, 0.6087436676025391, -0.2644941210746765, -0.11482396721839905, -1.097501277923584, -1.6510484218597412, 0.20077243447303772, 1.0244815349578857, -0.7942622303962708, -0.2956012487411499, -0.11457809805870056, -1.1713297367095947, 0.8365135192871094, -0.47443756461143494, 0.9144616723060608, 2.1477813720703125, -0.4577523171901703, -1.2418230772018433, -1.5436235666275024, -0.817193865776062, -0.09638527780771255, 1.256735920906067, -0.8776392936706543, 0.3979979455471039, -0.6781551241874695], [-0.8082117438316345, -0.024012308567762375, 0.8229308128356934, 1.1577116250991821, 0.16448664665222168, 1.58594810962677, 1.03529953956604, 0.6593875288963318, 0.38469991087913513, -0.05447527393698692, -1.0472307205200195, 1.6426488161087036, 0.21696950495243073, 0.6044728755950928, -1.1336621046066284, 0.09852644056081772, 0.07480733841657639, 0.6854264736175537, 0.28234240412712097, 0.7470940351486206, 0.6232008337974548, -1.6307694911956787, -0.2436329424381256, 1.8415677547454834, 0.5529549717903137, -0.3278525173664093, -1.1732162237167358, 0.3031913638114929, 0.04119335114955902, 0.3703983724117279, 0.09526193141937256, -0.5261039137840271, -0.13840922713279724, 2.867384672164917, -1.913848638534546, -1.3382220268249512, -1.011305570602417, 1.4932881593704224, -1.0406535863876343, -0.5520442724227905, -0.6239724159240723, 0.715086042881012, -0.12109432369470596, 1.7345426082611084, 0.10116619616746902, -1.81583571434021, 0.593413770198822, 0.11806793510913849, 0.21726322174072266, 0.26299184560775757, -1.8819894790649414, -0.16892381012439728, -0.2292792946100235, -0.6141343712806702, 0.6805010437965393, -0.1746576875448227, 0.7322977185249329, -0.47229185700416565, -2.5360636711120605, -0.5544314980506897, -0.9302642941474915, 0.454572856426239, -0.1215200275182724, 0.3756931722164154], [-0.23904076218605042, 0.20481663942337036, -0.38501521944999695, 1.3764134645462036, -1.1221102476119995, -0.6191027164459229, -0.40532010793685913, 1.5757637023925781, -0.01771635375916958, 1.6858690977096558, 1.533639669418335, 0.2062802016735077, -0.3276091516017914, -0.48448529839515686, -0.5217694044113159, 0.744185745716095, -1.213545799255371, -1.1971486806869507, 1.5715481042861938, 0.1342376470565796, 0.450619101524353, -0.3218540847301483, -0.5431699752807617, 1.7937290668487549, -0.8359166383743286, 0.9698861241340637, -3.8086636066436768, -2.008192777633667, 0.464983731508255, 0.3160821199417114, -1.539260745048523, 0.6623609662055969, 1.3419184684753418, -1.9339802265167236, 0.3205845057964325, 1.0088261365890503, -0.07040175795555115, -1.1580877304077148, 1.8233543634414673, 1.343934416770935, -0.18026478588581085, -0.8192018866539001, 0.09533119946718216, 2.058180570602417, 1.1983152627944946, 1.4668854475021362, -0.14939987659454346, 1.6621609926223755, -0.0923047587275505, -2.0610809326171875, 1.1334612369537354, 0.9939407706260681, 0.18096493184566498, 2.1304314136505127, 0.033885929733514786, -1.4898210763931274, -0.1931990087032318, -1.4351640939712524, 0.7762638330459595, -1.2425647974014282, -0.12687209248542786, 0.7519720792770386, -2.415088176727295, 0.2908537983894348], [1.4398224353790283, 2.178178310394287, -1.3316845893859863, 0.5256356596946716, -1.287534475326538, 1.2353845834732056, 0.5550066232681274, -0.12122270464897156, -0.8719409108161926, -2.388619899749756, 0.5177003145217896, 2.1732072830200195, 1.0472122430801392, -0.6181548237800598, 0.6903041005134583, -0.10267741233110428, -0.36134281754493713, 0.8431956171989441, -0.5392850041389465, 0.2210792899131775, -1.6034029722213745, 1.683426856994629, 0.3936213552951813, 0.39897456765174866, -0.2875571846961975, 1.8275766372680664, -0.28621092438697815, -0.09760603308677673, 3.585826873779297, -0.48820894956588745, -0.8423492908477783, 1.6850868463516235, 0.4534027874469757, 0.9088565111160278, 0.5138489007949829, -0.4782737195491791, 0.3084871172904968, -1.0791915655136108, -0.6203473806381226, -1.1968497037887573, 0.6751473546028137, -1.260186791419983, 0.4982629418373108, -1.410475730895996, -0.6571620106697083, -0.9808777570724487, 1.5515735149383545, 0.6032838225364685, -2.4448981285095215, 2.2979679107666016, -0.4282935857772827, 1.9593923091888428, 0.2912151515483856, -2.596606492996216, 0.8012544512748718, -0.5200638771057129, 0.18040013313293457, -0.19757197797298431, 0.45057713985443115, -1.342616081237793, 0.4047524929046631, -0.2833317816257477, -0.702742338180542, -0.25569555163383484], [-1.6450685262680054, 0.42643237113952637, -0.45124995708465576, -0.9056113362312317, 0.3503274619579315, 1.1899999380111694, -0.48528391122817993, 0.11848989129066467, -0.33377474546432495, 0.5653544068336487, -1.0590251684188843, 0.3828376531600952, 0.13609224557876587, 0.39153388142585754, 0.20402397215366364, -0.00023606274044141173, -2.178881883621216, -0.654687762260437, -0.1142694428563118, 0.19305214285850525, -0.25642964243888855, -0.24836422502994537, -0.21722795069217682, 0.5516858100891113, 1.8515156507492065, -0.4903978109359741, -1.4015529155731201, -2.114903211593628, -0.34681248664855957, -1.279188632965088, -0.4294966161251068, -1.2641438245773315, -0.3319914638996124, -0.11115933954715729, -0.42687371373176575, 0.774735689163208, 1.1124413013458252, -1.4081056118011475, 0.2326776534318924, 0.5334357023239136, -0.6423662304878235, -1.7324126958847046, -0.7811927199363708, 0.39010441303253174, 1.3363101482391357, -0.2749860882759094, 0.7960062026977539, -0.8005538582801819, 0.49976497888565063, 0.34387797117233276, 0.5964547395706177, -1.006718397140503, -0.4486329257488251, 0.5644047856330872, -0.005323314107954502, 1.6731480360031128, -1.9856370687484741, 1.3382569551467896, 1.2040009498596191, 1.1786551475524902, -1.2111133337020874, 0.21542499959468842, 1.3043484687805176, 1.2925183773040771], [0.16194984316825867, 0.5397354960441589, -0.18801285326480865, 0.06329726427793503, -2.6092028617858887, -0.5357838869094849, -0.09150028228759766, -0.7255094051361084, 0.33875682950019836, 0.37001800537109375, -2.094406843185425, -0.4920361042022705, -0.3939150869846344, 1.0928443670272827, -0.6577251553535461, -2.0055832862854004, 0.22729167342185974, -0.28076639771461487, -1.1962244510650635, -0.8223751783370972, 0.08180096000432968, 0.4892852008342743, -0.26944977045059204, -1.0554550886154175, 1.0464054346084595, 1.2212986946105957, -0.5507232546806335, -0.18334197998046875, 0.919975221157074, 0.11012884229421616, 2.0099377632141113, 1.006225824356079, 0.6879531741142273, 0.10843464732170105, -0.6581779718399048, 0.8004999160766602, -0.6858447790145874, -2.561358690261841, -1.508156180381775, -1.2375295162200928, 2.582618474960327, -0.48910123109817505, -1.3045777082443237, 2.0205953121185303, -0.7600414752960205, 0.9402212500572205, -0.7011981010437012, 0.15262672305107117, 0.12682777643203735, 0.0015222260262817144, 0.6262742280960083, 1.8885223865509033, -0.7373199462890625, -0.981472373008728, 0.24876238405704498, 1.2631876468658447, 0.8524923920631409, 1.3409764766693115, -0.6267359852790833, -1.156177043914795, 1.4271003007888794, -0.4631856083869934, -0.8241167068481445, 1.4750182628631592], [-1.7864782810211182, -1.5424078702926636, -0.159047469496727, 0.7247254848480225, 0.6596230864524841, -0.9825888872146606, 0.00040534252184443176, 0.09639150649309158, 1.262977957725525, 0.0014277093578130007, -1.417852759361267, 0.4381482005119324, -0.08148638904094696, -0.6847301125526428, -0.2705153524875641, 0.19748255610466003, -1.0371605157852173, 0.7707954049110413, -0.9386774301528931, 0.28529566526412964, -1.7656499147415161, 0.5944854617118835, -1.0359326601028442, 0.2397935837507248, 1.3012886047363281, -1.3036444187164307, -0.6553264856338501, -1.3287733793258667, -0.8459720015525818, 0.4894387722015381, -0.7230714559555054, -0.6367509365081787, 1.3917311429977417, 0.8483863472938538, 1.2109730243682861, -0.37727418541908264, 1.5482364892959595, 0.8501381874084473, -0.9437279105186462, -0.72249835729599, -0.2380443811416626, -0.3472420275211334, 0.3980841338634491, -0.20883449912071228, 1.8784428834915161, 0.669530987739563, 0.3931088447570801, -0.6181380748748779, -0.24733655154705048, 0.1475762128829956, -0.24123139679431915, 1.8990111351013184, -0.9534431099891663, 1.4214617013931274, -0.6228209733963013, 0.21911419928073883, -0.8378612995147705, -0.3799097239971161, -1.3944910764694214, -0.9451960325241089, 1.3270163536071777, -0.35884812474250793, -0.06769023090600967, 1.0489898920059204], [-0.4672633707523346, -1.1663751602172852, -0.2103421837091446, 1.7120604515075684, -0.6044716835021973, 0.11358693242073059, 0.974017858505249, 0.7175988554954529, -0.4526219964027405, 0.1096540316939354, -0.302501380443573, 0.16784092783927917, -1.0136454105377197, 0.7150171995162964, -0.17387796938419342, -0.6957963705062866, -0.7196743488311768, 0.6347883343696594, 0.3117932975292206, 1.5951259136199951, 0.42431798577308655, 0.12578806281089783, 1.0865811109542847, 1.3640635013580322, 1.4981372356414795, 0.6336464285850525, -0.9718031883239746, 1.2571747303009033, -0.6085721254348755, 0.4904060661792755, 1.1702316999435425, -0.8562440276145935, 0.9494125843048096, 1.5021626949310303, -0.40324532985687256, 0.49216556549072266, -0.28694581985473633, -1.788676381111145, -1.0355098247528076, 0.6250664591789246, -0.04931548237800598, 0.5535362958908081, -0.07709124684333801, 0.3947378098964691, -0.7645958662033081, -1.0830111503601074, -0.1395440399646759, -0.07073463499546051, 1.9234815835952759, -0.5102875828742981, -1.4880375862121582, -0.482799768447876, 0.02058013714849949, 0.4787280857563019, -0.3016511797904968, 0.11117285490036011, 0.6547031998634338, 1.8152194023132324, -1.2316347360610962, 0.917996883392334, 0.013574252836406231, 1.1828699111938477, 0.5860446691513062, -1.071664810180664], [-0.49300912022590637, 1.1198915243148804, 2.913353443145752, 0.313427597284317, 1.865246057510376, -0.4242466390132904, -0.8966554999351501, -1.5197733640670776, 1.1669930219650269, 0.9672468304634094, -2.402759075164795, 0.11829080432653427, -0.2020290046930313, 0.6413363218307495, 0.9538189768791199, 0.6948491930961609, 0.9161077737808228, -1.5348752737045288, 0.8625762462615967, 0.9568737745285034, -0.17060621082782745, 1.734948992729187, -0.8929805159568787, 0.42401963472366333, 0.17451126873493195, -0.20257768034934998, -0.5580076575279236, -0.9123759269714355, -0.22403942048549652, -0.5557135939598083, -0.4816040098667145, 0.37586236000061035, -0.5029045343399048, -0.13273213803768158, -0.32825034856796265, 0.9004512429237366, 0.4735018014907837, -0.2389698475599289, -1.8632892370224, 0.0790402963757515, 1.112295389175415, 2.3124897480010986, -1.1589887142181396, 0.09689871221780777, -0.15069785714149475, 0.3271300494670868, -1.0532426834106445, -2.604687213897705, 1.2242499589920044, 1.1410086154937744, 0.25511980056762695, -0.7557850480079651, 1.270114779472351, -1.0064244270324707, 0.04979193955659866, -0.6122106909751892, -0.6605513095855713, -0.1874532401561737, -0.12558533251285553, -0.24474374949932098, -0.20474529266357422, 0.0028224466368556023, -0.33142387866973877, -0.2762407064437866], [0.03244875743985176, -0.7496687173843384, -0.01552653405815363, -0.13522455096244812, 0.8153500556945801, -0.1674797534942627, -1.4566216468811035, -0.01501296553760767, 1.0963648557662964, -0.28057023882865906, 0.12747828662395477, -0.9352880120277405, -0.7930359244346619, -0.6926189661026001, 0.18003389239311218, 1.5312566757202148, -0.6127782464027405, -0.7107869386672974, 1.3132884502410889, -0.36570173501968384, 0.7513702511787415, 1.349034309387207, -0.7340747117996216, -0.3081636130809784, 0.79465252161026, -1.1639938354492188, 0.5316853523254395, -0.6916162371635437, 0.8498539328575134, 1.0829699039459229, 1.0598318576812744, 0.5451200008392334, 0.05846302583813667, -0.3390346169471741, 0.39267468452453613, -1.331743836402893, 1.295811414718628, -1.3348770141601562, -0.941599428653717, -0.9752289056777954, -0.20887504518032074, 0.083188496530056, -1.0618518590927124, 0.3304333984851837, -0.3589833974838257, -1.299765706062317, 1.3913962841033936, 0.6428854465484619, -0.8171914219856262, -0.24322792887687683, -0.934899628162384, -0.2535635530948639, 2.2433605194091797, 1.1371839046478271, -0.3359513282775879, -0.003408940741792321, 0.4388265907764435, 1.0414824485778809, -0.01261176448315382, -1.131628394126892, -0.2960793375968933, -0.23095789551734924, -0.1804986149072647, -0.7585195302963257], [-0.4887505769729614, 1.013426423072815, 1.0250608921051025, 1.4527159929275513, -0.44713571667671204, -0.6978197693824768, -0.0730501264333725, 0.730572521686554, 1.3893758058547974, -0.6199505925178528, 0.09802458435297012, -0.31061142683029175, -0.7183239459991455, -3.3333990573883057, 0.6581125855445862, -0.7674590945243835, -0.2365717887878418, -0.19292117655277252, 0.15025247633457184, -0.684627115726471, 0.6852529048919678, 0.6353568434715271, -0.674978494644165, 0.4684523940086365, -0.24706301093101501, 0.7754725217819214, -1.1107991933822632, 0.7255873680114746, -1.3706220388412476, -1.0052400827407837, -0.6845162510871887, 0.22177761793136597, -0.12765182554721832, 0.17583878338336945, 0.4001908302307129, 1.6748343706130981, 0.25037881731987, 1.5814611911773682, 0.009603405371308327, -1.1522190570831299, 1.8689119815826416, 0.09958745539188385, 0.2833390533924103, -0.47324126958847046, -0.1334136426448822, -0.6410586833953857, 0.06948978453874588, -1.5675950050354004, 0.14001579582691193, -0.2634921371936798, -0.5082928538322449, 1.5238454341888428, -0.24704709649085999, -0.7374048829078674, 1.148390769958496, 1.0730897188186646, -0.008295120671391487, 1.6630750894546509, 0.20665523409843445, -0.0594400092959404, 2.424356460571289, -0.5948176383972168, 1.4263032674789429, -1.4966278076171875], [-0.8842142820358276, 0.02725045382976532, 0.40991538763046265, -0.5303919315338135, -0.5386804938316345, 0.989403486251831, 0.20506848394870758, 0.2578057646751404, 0.7104113698005676, 0.10846208035945892, -0.29155659675598145, 0.9207481145858765, -1.0253347158432007, 0.11160878092050552, -1.7094922065734863, -2.4892940521240234, -0.3524981141090393, -0.3645057678222656, -0.7574055194854736, -0.9525278806686401, 0.09215589612722397, 0.6119401454925537, -0.8575329184532166, 1.9632749557495117, -0.8393318057060242, -0.36885538697242737, -0.5499789714813232, 0.10299735516309738, -0.9744760990142822, 1.3534704446792603, 1.7445212602615356, 0.6237545013427734, 2.9702162742614746, 0.3875653147697449, 0.45725852251052856, -1.3886865377426147, -0.9976429343223572, -0.6984962224960327, 0.47811105847358704, -0.10148625820875168, -0.23925255239009857, 0.10317666083574295, 0.5516418218612671, 0.3595316410064697, 0.09096982330083847, -1.490219235420227, -1.2936381101608276, 1.2199510335922241, 1.9333686828613281, -0.025881750509142876, 0.11284536868333817, 0.1545257568359375, -1.071042537689209, -0.020565729588270187, -0.8473731875419617, -0.1569986641407013, 0.34129634499549866, -0.07717114686965942, 0.5859916806221008, -0.5141134262084961, 0.5608969330787659, -1.1862030029296875, -1.5605493783950806, -0.13834993541240692], [-0.38904276490211487, 0.0474376305937767, 0.17472849786281586, -0.5638571977615356, -0.415218710899353, -0.638927698135376, -0.8372513651847839, -0.7592064142227173, -1.1454654932022095, -0.8678951859474182, 0.08386576175689697, -1.8620065450668335, -1.6862926483154297, -0.7966267466545105, 0.015250113792717457, 0.8421606421470642, -0.3505856990814209, -0.910775363445282, 0.1592579483985901, 1.2824339866638184, -0.3461230993270874, 0.7163034677505493, 1.0368297100067139, -0.3290518522262573, -0.4387196898460388, -0.13553351163864136, 0.33985060453414917, -1.0943787097930908, 1.4379793405532837, -0.01564466953277588, 0.15456144511699677, 0.644212543964386, -0.050081223249435425, -0.643003523349762, -0.7378391623497009, -0.579990029335022, -1.1723512411117554, -0.4834814965724945, 0.16860529780387878, 0.10266391187906265, -0.564176619052887, 0.22778232395648956, -0.46895983815193176, 0.726324200630188, 1.5050408840179443, -0.9051281213760376, 0.8940613865852356, 0.35069918632507324, -0.1724519580602646, 0.5784409046173096, -1.0043916702270508, 0.8347668051719666, 1.813252329826355, 1.1599531173706055, 0.35873880982398987, 1.7752013206481934, 2.111980438232422, -2.0243916511535645, 0.0477922186255455, -0.3763636648654938, 0.4480549693107605, 0.4766969680786133, -0.7972798943519592, 0.31035831570625305], [-0.47739502787590027, 0.3027113974094391, -1.0980976819992065, -1.9231534004211426, -0.7220457196235657, 1.691001057624817, -0.5264744758605957, -0.31712520122528076, 1.3371105194091797, -0.594802975654602, 0.2198549062013626, -1.5281544923782349, 1.4527486562728882, -0.242500439286232, -0.29153627157211304, 1.4552973508834839, 0.07880546152591705, -0.14320108294487, -1.1118813753128052, -0.9098613858222961, -0.6458548903465271, 0.4342040419578552, -0.20713096857070923, -0.09965316206216812, -0.04476102069020271, -0.515144944190979, -0.2573888599872589, -1.486158013343811, 0.316303014755249, 1.0373369455337524, 0.38137295842170715, -2.439389944076538, -0.5207028388977051, -1.0190545320510864, -0.03921826183795929, 0.3007766306400299, 1.7749013900756836, -0.2965472340583801, -0.310838520526886, -0.42753365635871887, 1.414017677307129, 1.4003214836120605, 0.5049219727516174, -0.1277041733264923, -0.12160620093345642, -1.4290186166763306, -0.020128563046455383, 0.5863465666770935, -1.5543760061264038, 1.1118426322937012, 0.5313984155654907, -2.3339297771453857, -0.16111011803150177, -1.1203510761260986, -0.808623731136322, -0.27191948890686035, -0.10148708522319794, -0.1866849809885025, 0.7182176113128662, -0.9442884922027588, -1.67203688621521, 0.752234935760498, -1.617750644683838, 0.7332345843315125], [1.0138766765594482, 0.6808452606201172, 0.011526353657245636, -1.4804518222808838, 0.484123557806015, 1.2355016469955444, 0.8357691168785095, 1.562574028968811, 0.47826874256134033, 2.3053441047668457, -1.220429539680481, -1.0789637565612793, -0.7741320133209229, 0.5000130534172058, -0.527168333530426, -0.9956972002983093, 0.32769888639450073, -0.2872711420059204, 1.6656887531280518, -1.9741038084030151, -0.7210201025009155, -0.6463971138000488, -0.9572387933731079, -0.22678571939468384, -1.066659688949585, 0.037749484181404114, -0.47072941064834595, -1.1990336179733276, 0.2169618010520935, 0.9201063513755798, 0.3464702367782593, 0.1813536435365677, 0.4912888705730438, -0.0133588295429945, -0.036106936633586884, -3.187786817550659, 0.8307927846908569, 0.42655816674232483, 0.23516978323459625, 0.48490896821022034, -1.9143296480178833, 1.1583948135375977, -2.185270071029663, -1.4456164836883545, 1.6410548686981201, 0.3363293707370758, -1.7196084260940552, 0.16997571289539337, -0.027717074379324913, -0.1480424553155899, -0.024689925834536552, -2.2574973106384277, -1.1573735475540161, -1.444685697555542, -1.2727762460708618, -0.7488813996315002, 2.2515320777893066, -0.6260969638824463, 0.6165204644203186, 0.6121826767921448, -0.22801005840301514, 1.1769450902938843, 0.1992611289024353, -0.726578414440155], [-0.634712815284729, 1.0024394989013672, 0.9582006335258484, -0.859830379486084, 0.02030950039625168, 0.7747251987457275, -0.9665512442588806, 1.222649335861206, -0.8173168897628784, -0.995667040348053, 0.9056174755096436, -0.5679169297218323, 1.6583906412124634, -0.9803639054298401, 1.5048996210098267, 1.7706172466278076, 0.11808713525533676, -1.3912248611450195, -0.27714669704437256, -1.1205195188522339, 0.29269999265670776, 0.24727530777454376, -0.060336291790008545, -1.2371318340301514, 1.3503135442733765, 0.3303054869174957, 1.8142234086990356, 1.6562660932540894, 0.48657745122909546, -0.2723056972026825, 0.20925448834896088, 0.6924271583557129, -0.7677631378173828, -0.5865437984466553, -1.5579239130020142, 1.27581787109375, 0.38283106684684753, 0.2808479964733124, -2.9188785552978516, -0.6662977933883667, 0.9191856384277344, -0.24141749739646912, 1.401219129562378, -2.0589592456817627, 0.9301231503486633, 0.24338442087173462, 0.9784338474273682, 1.7509629726409912, 1.4238606691360474, -0.0949196070432663, -0.2105400115251541, -1.030394196510315, 0.48738405108451843, -0.15963035821914673, -0.11402064561843872, 0.032393429428339005, 0.30755746364593506, -0.8590268492698669, 1.0855612754821777, -0.06606201082468033, -0.33534738421440125, 0.5065895915031433, 0.5799367427825928, -1.4666029214859009], [1.2819827795028687, 0.34972330927848816, -1.2524763345718384, 0.05111318454146385, -0.9415059089660645, 1.1615040302276611, 1.3789441585540771, -0.5767348408699036, -0.9889910221099854, -0.7962681651115417, -1.2816864252090454, 0.07210789620876312, -1.7837097644805908, -0.14489662647247314, -1.19416081905365, -0.16911442577838898, 0.054958563297986984, -0.09947721660137177, -0.5141138434410095, 2.0509836673736572, 0.12487421929836273, -0.3512837588787079, -0.15867659449577332, 0.7562960982322693, 1.9284919500350952, -0.7983083724975586, 0.3514593541622162, -0.5793306231498718, -0.16500161588191986, 1.6184895038604736, 0.9268848299980164, -1.0046652555465698, 0.6772757172584534, -0.13268302381038666, 0.24676446616649628, 0.7272411584854126, -0.9416292309761047, 0.2667982876300812, 1.5232794284820557, -0.09967371076345444, 1.7675223350524902, 0.8820798993110657, -1.0636998414993286, -0.9965896606445312, -0.11761543899774551, -0.7255882620811462, 0.024263447150588036, -0.37940263748168945, 1.7168232202529907, 1.7434927225112915, -0.556124746799469, -0.7571262121200562, 1.649420142173767, 0.295236736536026, 0.9575192928314209, 0.9068781137466431, -1.0832324028015137, 0.22486259043216705, 0.11891684681177139, -1.4460912942886353, -0.8101760745048523, 0.2951512038707733, 0.3944167196750641, -0.16478261351585388], [-1.3263493776321411, 0.5525830388069153, 2.53910756111145, 0.12660804390907288, -0.14034061133861542, -0.431959867477417, -1.0542834997177124, 0.1395823061466217, -1.6533691883087158, -1.7787235975265503, -1.707281470298767, 1.0838596820831299, -0.2727741301059723, 0.8836604952812195, -2.0362398624420166, -1.2594813108444214, -0.5086968541145325, -1.2218568325042725, -0.936426043510437, 0.36086103320121765, -0.6884899735450745, 0.19163401424884796, 1.07249116897583, 0.8563118577003479, 1.277488112449646, -1.0852265357971191, 0.5929372310638428, 1.8202457427978516, 1.6061311960220337, 0.7651886940002441, -0.016745634377002716, 0.705217719078064, -0.18091179430484772, 0.4400164484977722, 1.5104438066482544, 0.026631362736225128, -1.1509593725204468, 1.2261606454849243, 1.2340425252914429, 0.45467454195022583, -0.11600513756275177, 0.030645903199911118, -1.36367666721344, -0.6073353290557861, -1.9060126543045044, -1.0036581754684448, -1.5010101795196533, -1.7420861721038818, -1.7865976095199585, -0.1283433437347412, 0.23673950135707855, -1.0495765209197998, 0.4568277895450592, 0.5043716430664062, 0.12277455627918243, -1.1398751735687256, 1.0130674839019775, 0.22572821378707886, -0.46221187710762024, -0.22083882987499237, 0.13234291970729828, 1.5472307205200195, -0.49199074506759644, 1.3361520767211914], [-1.3729970455169678, 1.1581792831420898, -1.2272504568099976, -0.6253589987754822, 0.46513622999191284, 0.3656788170337677, -0.3852510452270508, 1.0846656560897827, -0.5656628012657166, -1.0860792398452759, 0.7917559742927551, -1.6025336980819702, -0.22201970219612122, -0.7217912077903748, -2.216442346572876, 0.4578009247779846, -0.09253554046154022, -1.9279959201812744, 0.4186357259750366, 0.006562521681189537, -0.6694725155830383, -1.3801318407058716, 0.7588515281677246, 1.247793436050415, 0.3033786416053772, -0.4758046269416809, -0.9122676253318787, 0.15754766762256622, 0.16050896048545837, 1.1208547353744507, 0.14356684684753418, 1.6027432680130005, -0.6097273230552673, 0.5760367512702942, 0.2534213364124298, 1.1277140378952026, -0.7236465215682983, 1.457576870918274, 0.31502965092658997, -0.4368171989917755, -0.6223031878471375, 1.9709396362304688, 0.5569257736206055, -1.5084882974624634, 0.143728569149971, -0.6861610412597656, -1.606592059135437, 0.44418373703956604, -0.7501295804977417, -0.5197678208351135, -1.8498163223266602, 1.4673690795898438, 2.28426194190979, -0.9221250414848328, 0.24870207905769348, 1.403505802154541, 0.3916318416595459, -0.5537561178207397, -1.6652361154556274, 0.971942126750946, -0.5503974556922913, -0.83207768201828, -0.5636944770812988, 0.22990602254867554], [0.8770689368247986, -0.2988189160823822, 1.8604869842529297, 0.20524565875530243, -1.2859477996826172, 0.6881566047668457, 0.32678142189979553, -1.163940191268921, 0.11980514228343964, 2.061870813369751, -1.1579997539520264, 0.47956302762031555, -1.134318232536316, 1.2530544996261597, -1.3211861848831177, 0.6054027080535889, -0.09943652898073196, -1.0256356000900269, 0.4770936369895935, -0.377515971660614, -0.003910287283360958, -0.7989808917045593, 0.9118232727050781, -1.0945312976837158, 0.6480805277824402, -2.201885938644409, -1.1343967914581299, 0.4758298695087433, 0.8553482890129089, -1.940200686454773, -1.4750275611877441, -0.8779138326644897, -1.604297161102295, -1.3494045734405518, -1.1815154552459717, 1.7991838455200195, -0.9570057988166809, 0.42700374126434326, -0.7446187138557434, -0.9882619976997375, -1.8840409517288208, 0.7865239977836609, 1.61600923538208, -0.4342361390590668, -1.1507647037506104, 0.653404712677002, -0.10890951007604599, 0.6668485999107361, 1.4714446067810059, 0.21721981465816498, 0.8103966116905212, 0.04949566721916199, 0.720338761806488, 0.6634928584098816, -1.368119716644287, -1.0096856355667114, 1.2756948471069336, 0.4828624427318573, 0.7474512457847595, 1.7557376623153687, 1.5138227939605713, -0.4701913297176361, -0.5785232782363892, -0.17213474214076996], [-2.803422212600708, -0.30589601397514343, 1.045994520187378, 0.5593043565750122, -0.8996959328651428, 0.697895884513855, 2.169387102127075, 1.2194721698760986, -0.34661492705345154, 0.054197534918785095, -0.06193196773529053, -0.5747779011726379, 1.045366644859314, 0.2335406094789505, 0.5914279818534851, 0.49646833539009094, 0.2732243239879608, 2.1153581142425537, -0.4035482108592987, -0.06452430039644241, 0.5971193313598633, 0.7901395559310913, -1.1159636974334717, -0.06409570574760437, 0.878897488117218, -0.8430097103118896, -0.22339168190956116, -0.9442647099494934, -1.9672318696975708, 1.0077457427978516, 0.835328996181488, 0.1542191505432129, -0.43839746713638306, -0.2535090148448944, -0.554658830165863, 0.1894284188747406, 0.22396618127822876, 1.2351748943328857, -0.03456360101699829, -0.3858075737953186, -0.3599463403224945, 0.5862454175949097, 1.916846513748169, 1.782415747642517, -0.6854515671730042, -0.8147256374359131, 0.453806608915329, -0.24080204963684082, 1.0132871866226196, 0.7692386507987976, -0.7290489673614502, 0.658806324005127, -1.0663056373596191, -0.9434237480163574, -0.2050473839044571, 1.6068916320800781, -1.5096007585525513, 0.04227185621857643, -0.22539053857326508, -0.23654012382030487, -0.34506916999816895, 0.2130681425333023, -0.6229602694511414, -0.43135932087898254], [-0.8984630107879639, 1.1389988660812378, -0.2251524031162262, -0.04995490238070488, -1.6252559423446655, -1.6148172616958618, -0.05732358247041702, -1.3360629081726074, 2.4869635105133057, 0.03297639265656471, 0.9668936133384705, 0.5203155875205994, -1.2813299894332886, 0.04575103521347046, 0.6897023916244507, -0.44391995668411255, -0.9924116730690002, -0.18613675236701965, -1.5811779499053955, 2.09081768989563, -0.822447657585144, -0.6979201436042786, -1.1174745559692383, 0.683819055557251, 0.945041835308075, 0.6031186580657959, 0.7989274263381958, 0.06607592850923538, -1.6397677659988403, 0.25311437249183655, -1.110422134399414, -0.8558861613273621, 0.03786671534180641, -0.6229466199874878, -0.09341899305582047, 0.8497776389122009, -1.315443754196167, -0.19533796608448029, -1.1922117471694946, 0.6906496286392212, 1.1727937459945679, -0.5781371593475342, -1.247245192527771, 0.7404548525810242, -0.38807839155197144, -0.21346546709537506, -0.6567530632019043, 0.5524293184280396, 0.16188722848892212, 1.2029812335968018, -0.3129749894142151, -0.8127081394195557, -0.3445190489292145, -0.9561517834663391, 0.22511819005012512, -1.222771167755127, 0.8174541592597961, -0.4519839584827423, -0.11152340471744537, 0.45212242007255554, -0.667184591293335, 0.5889460444450378, -0.46225517988204956, -0.3643110394477844], [-0.45068269968032837, -0.9940283298492432, -0.4543386697769165, 0.40961262583732605, -0.5126320719718933, -0.4862525463104248, 1.5263831615447998, 1.1153920888900757, 1.2040334939956665, 0.9791149497032166, -0.2665835916996002, -0.07668087631464005, 0.21526530385017395, 1.3764537572860718, 0.44328153133392334, -0.2612592875957489, 0.13094119727611542, -0.3661476671695709, -0.17255087196826935, 0.5343234539031982, -1.3910479545593262, 0.605247974395752, 1.1447826623916626, -2.3475329875946045, -0.03848152235150337, 3.0434205532073975, 0.39810293912887573, 0.8126767873764038, -0.19940617680549622, -0.7770474553108215, 0.6903774738311768, -0.9443444609642029, -0.06970786303281784, -0.6182120442390442, 0.984493613243103, 1.0165709257125854, 0.916273832321167, 0.9133129715919495, -1.1517903804779053, -0.18697261810302734, -1.8622708320617676, 0.7338275909423828, 0.27574723958969116, -1.1164300441741943, -1.2600317001342773, 0.03309224918484688, -1.1139732599258423, 2.0589520931243896, 1.0514739751815796, -0.37865856289863586, -0.7053806185722351, 0.9929459691047668, -2.040534019470215, 1.6700773239135742, -0.9310763478279114, -0.43131548166275024, 0.7493193745613098, -0.20910798013210297, 0.04793458804488182, -0.48356202244758606, -0.17657479643821716, 0.7019301056861877, 0.7009309530258179, -1.5443087816238403], [1.215738296508789, 1.2503292560577393, 1.6419328451156616, 0.09346721321344376, -0.41102319955825806, -0.054668135941028595, -0.16757595539093018, 0.33007562160491943, -0.9637336730957031, -1.9860788583755493, 0.28243550658226013, 0.16794057190418243, 0.5946502089500427, -0.3406098783016205, -1.0074712038040161, -0.06558398902416229, -0.4123137891292572, 0.20638464391231537, 1.32008957862854, -0.38885191082954407, 0.02593495137989521, 0.9723391532897949, -0.14536459743976593, 0.6245831251144409, 0.9811461567878723, -0.9648062586784363, 0.5382890701293945, -0.677547812461853, -1.2830908298492432, 0.2136053442955017, -2.360654592514038, 0.656663179397583, -1.3451019525527954, 0.8789694309234619, 0.1673882007598877, -0.5208109021186829, 0.633215606212616, -0.6357025504112244, 2.638805866241455, -0.3351408839225769, 0.2898799180984497, 0.5431475639343262, 0.21262003481388092, -0.5923975706100464, -0.6536708474159241, 1.110032558441162, -0.9095306992530823, 0.8087185025215149, -1.9913785457611084, -0.369666188955307, -1.9409679174423218, -2.598144292831421, 0.4745587706565857, 0.34315383434295654, -1.6750613451004028, 0.6891628503799438, 0.39483413100242615, -1.6249372959136963, 1.2130661010742188, -1.029429316520691, 1.6592053174972534, 0.3938997685909271, 0.49106666445732117, -0.15697550773620605], [1.6269547939300537, -0.3435351550579071, -0.3881143033504486, -0.11196742951869965, -0.9502397179603577, -1.0462305545806885, 0.11112295091152191, -0.1239176094532013, 0.48339545726776123, 1.141937255859375, -0.23325560986995697, 0.26424744725227356, -0.37001466751098633, -0.4104978144168854, 0.6155555248260498, -1.170088768005371, 0.1967899203300476, 0.16770166158676147, 0.07178094238042831, -0.41613632440567017, -1.5980079174041748, -1.5782886743545532, 0.03695949539542198, 0.2090425044298172, -0.03519296646118164, 1.5646072626113892, 0.3560132384300232, 2.1581878662109375, 1.7145572900772095, -0.5194740891456604, -0.30198025703430176, 2.6603188514709473, -1.3696163892745972, -0.6984050869941711, -0.4138679802417755, -0.843802809715271, -0.22480152547359467, 0.019876405596733093, -1.497530460357666, -0.014448339119553566, 0.178681880235672, 0.5205232501029968, 0.5745465755462646, 1.0635758638381958, 0.7757693529129028, -0.6685522198677063, 0.5135119557380676, 2.065814971923828, 0.7003605961799622, -0.6294448971748352, -0.43063971400260925, -0.8257330656051636, -0.3597636818885803, -0.840861976146698, -1.1119567155838013, -0.7063302397727966, -1.67406165599823, 0.2851148545742035, 1.1329281330108643, 1.525956392288208, -0.5608044862747192, 1.972542405128479, -1.1181323528289795, 1.2650688886642456], [-0.7249823212623596, 0.26394787430763245, 0.0403219535946846, -0.6296319365501404, -0.30169034004211426, 0.551441490650177, -2.635512351989746, -0.3692958950996399, -0.2862907648086548, 0.4452070891857147, -0.3549639880657196, -0.1742180436849594, -0.019676491618156433, 1.0683149099349976, -0.028194662183523178, 0.23583242297172546, 0.07059678435325623, 1.9619964361190796, 0.8443505764007568, 0.017779305577278137, -1.0171713829040527, 0.4809412956237793, 2.3075194358825684, -0.4828687310218811, -0.7294420003890991, -0.6113547086715698, 1.2446156740188599, 2.331717014312744, 1.0161229372024536, 0.521358072757721, 0.026127882301807404, 0.5617991089820862, 0.1714363992214203, -1.0906201601028442, 0.8861850500106812, 0.12086209654808044, 1.14629065990448, 1.41220223903656, 0.007087105885148048, -0.9495665431022644, 1.323105812072754, 0.1644008457660675, 0.4622780382633209, -2.1499557495117188, 0.5132845640182495, 0.2877636253833771, 1.251328945159912, 1.1190143823623657, -1.252741813659668, -0.537055492401123, 0.6236209869384766, -0.19190062582492828, -0.2843480706214905, 2.291822910308838, 2.6305108070373535, 1.7340031862258911, -0.8125677108764648, -0.3248993456363678, -1.8746322393417358, 0.41233980655670166, -0.17597290873527527, 0.21512150764465332, -1.4118977785110474, -1.2550102472305298], [-0.8830357193946838, 0.6023883819580078, 0.7553927302360535, -0.490405410528183, -0.5863017439842224, -0.11783670634031296, 0.10105309635400772, 2.003936767578125, -0.8959711194038391, -0.47471708059310913, -0.1044904813170433, 0.04156581312417984, 0.3669959008693695, -0.2836263179779053, -1.2690980434417725, -0.8329979777336121, 0.4839256703853607, -0.24008245766162872, -0.4115917682647705, -0.13184848427772522, -0.6715887188911438, 1.5454844236373901, -0.8483806848526001, -1.40669846534729, -2.5633156299591064, -1.1366804838180542, 0.6133986711502075, -0.888793408870697, -0.4431757926940918, 1.223154067993164, -0.12916801869869232, 0.44974595308303833, 0.5272565484046936, -0.4570334255695343, 0.41686058044433594, -1.3018195629119873, -1.158075213432312, 1.8242079019546509, -0.6723418235778809, 1.2622061967849731, 0.6601359844207764, -1.175369381904602, 0.018824748694896698, -0.14944875240325928, -0.6323321461677551, -0.574781060218811, 0.531679093837738, -0.5603393316268921, 0.9755340218544006, -0.05782635882496834, 0.8115764856338501, -0.04427509382367134, -0.5503388047218323, -1.2086910009384155, 0.507239580154419, -2.6371102333068848, 0.1373177319765091, 1.6168941259384155, -0.8497002124786377, 1.411339282989502, 0.44505271315574646, -1.7614192962646484, -3.4325215816497803, 2.0963799953460693], [-0.33638808131217957, -0.9402511715888977, 0.9109178781509399, 0.33171918988227844, 0.5571764707565308, 0.21323946118354797, 1.537491798400879, -0.8857259154319763, 1.737288475036621, -0.26107218861579895, 2.42463755607605, 1.425607681274414, -0.7198894619941711, -0.5840480327606201, -0.2990352511405945, -0.34331029653549194, 0.48496782779693604, -0.5657998323440552, -0.4973281919956207, 0.5951301455497742, 0.13849298655986786, -0.32807499170303345, 0.8296424150466919, 1.958430528640747, 0.618577778339386, -2.3049416542053223, 0.814171314239502, -0.5476232171058655, 0.3841121196746826, -0.7795345187187195, 0.06039970740675926, 0.4804786443710327, 0.7437214851379395, -0.16237765550613403, 0.6555540561676025, -0.053544625639915466, -0.0802217349410057, -0.6480543613433838, -1.6283289194107056, -0.6406806111335754, 0.36933907866477966, -0.22264325618743896, -1.929248332977295, -1.2515385150909424, 1.9352672100067139, -1.756795048713684, -1.5336889028549194, 0.7528367042541504, -0.6107024550437927, 0.6418712735176086, 0.407647967338562, 1.1616995334625244, -0.41579878330230713, 0.4605903923511505, 0.08396957069635391, 0.03900105133652687, 1.4725533723831177, 0.6322447061538696, 0.6534271240234375, -0.20116634666919708, 0.19749632477760315, 1.9523314237594604, 1.4926021099090576, -0.4815923571586609], [0.8642063140869141, -2.3445050716400146, -0.9794037938117981, -0.3777692914009094, 0.9279014468193054, -1.3996449708938599, 0.7066558599472046, -1.475262999534607, 0.6716110110282898, 0.442800909280777, 0.2564544081687927, -1.1225121021270752, -0.25318941473960876, 1.3398311138153076, -1.473969578742981, 1.3020148277282715, 2.4365127086639404, 1.855802297592163, -1.0493433475494385, 0.9432471394538879, -1.1697818040847778, 0.7391439080238342, 0.12142106890678406, -0.501105785369873, -0.28810325264930725, -0.2978513240814209, -0.09712453186511993, 0.6487956047058105, -0.13471268117427826, 0.28498995304107666, 0.23987267911434174, -1.660470724105835, 1.4835485219955444, 0.7806585431098938, 1.3879872560501099, 0.6414170265197754, 0.23457999527454376, 0.9007160067558289, -0.37934938073158264, 0.23747536540031433, 2.0839896202087402, 2.599123477935791, 0.3021742105484009, 0.452818363904953, -0.07105283439159393, -0.27785512804985046, 0.6445685029029846, 0.7448015809059143, 0.5783746242523193, -0.6353228688240051, -0.7789162397384644, -0.7273075580596924, -0.33450329303741455, 1.582730770111084, 0.37937062978744507, 0.31781917810440063, -0.08817330747842789, 0.4608851969242096, 0.5917221307754517, -0.6349624395370483, 0.15102319419384003, 0.4262031018733978, 0.3255878686904907, 0.3551994860172272], [-0.0947573110461235, 0.4241291880607605, -1.7095026969909668, 0.3972886800765991, 0.7979140281677246, -0.8477243781089783, -0.07811855524778366, -0.1645481437444687, 1.1400904655456543, -1.2592881830642e-05, -0.39948827028274536, -1.0112779140472412, 0.8260934948921204, -1.2712721824645996, -2.4520411491394043, -0.22507202625274658, 0.093851737678051, -0.0046567153185606, -0.6443497538566589, 0.14103379845619202, 0.5759131908416748, 0.015106015838682652, 1.3913570642471313, -0.958722710609436, 0.01576686091721058, -0.9762833118438721, 1.296722650527954, -0.6862571239471436, -0.14792384207248688, 0.7570928335189819, -0.9498258233070374, -0.013688319362699986, -0.4625297486782074, 1.2163236141204834, -0.16921043395996094, 0.6085711717605591, -0.7314308881759644, 0.9969421625137329, 1.270622968673706, 0.5903156399726868, 1.197752594947815, -0.42284879088401794, 0.007254408672451973, 0.07141407579183578, -1.1542607545852661, -0.11549729108810425, 2.266227960586548, -0.8796970844268799, -0.23292943835258484, 0.4470374584197998, 1.9047341346740723, -0.6113829016685486, -0.14557121694087982, 0.5107647776603699, -1.7835191488265991, 1.4688034057617188, -0.6956354975700378, -1.1686712503433228, -0.5930685997009277, -1.4458413124084473, -0.19464297592639923, 0.3442600965499878, 1.3452280759811401, -0.39710861444473267], [-0.8598288297653198, -0.22089578211307526, 0.7809937596321106, 0.946194589138031, -1.1480761766433716, 0.59783536195755, 0.2306925654411316, -0.6288811564445496, -0.7508731484413147, -1.335161566734314, 0.5840030908584595, 0.3563016951084137, 0.416559636592865, -2.837251663208008, 0.7853622436523438, -1.6182715892791748, 0.4269525110721588, 0.7672380208969116, 0.5139446258544922, -0.7642852663993835, 0.5704686045646667, -2.072334051132202, -0.49336859583854675, 0.11345957219600677, 1.0484645366668701, 0.9802424907684326, 1.079638123512268, -0.3622499406337738, -0.6937612891197205, -0.1755535751581192, 0.43112680315971375, 2.070221424102783, -1.0691436529159546, 0.648720920085907, -0.046417221426963806, 0.22251589596271515, -0.3378904163837433, -0.32308635115623474, 0.10927660763263702, 0.4139671325683594, -0.7734287977218628, -1.8863763809204102, 0.02324868179857731, 0.5954208970069885, -0.4901484251022339, -2.2562971115112305, -2.266563892364502, 0.47612181305885315, 0.18790772557258606, -0.007560760248452425, -0.7221455574035645, 0.5408248901367188, -0.010331151075661182, 1.320233702659607, -1.0795506238937378, 1.2587831020355225, 1.3315695524215698, 0.26565247774124146, -0.435964971780777, 0.5727816224098206, 0.6129502654075623, -1.1842262744903564, -1.2536003589630127, 2.6236565113067627], [1.0468636751174927, -0.21139496564865112, -0.12892481684684753, -0.8768252730369568, 0.058930911123752594, -1.1231884956359863, -1.348556637763977, 0.15391160547733307, -2.0912718772888184, -0.9792670607566833, -0.04020986706018448, -1.067665457725525, 0.2397845834493637, 1.286539077758789, 0.07780967652797699, 0.3939913511276245, 0.1418265998363495, 0.077105313539505, -0.7971459031105042, -0.6627119183540344, -0.12176098674535751, -0.33992984890937805, 0.9885132312774658, 0.517596423625946, -0.6542061567306519, -0.5253685712814331, -0.4133802652359009, 0.38961365818977356, 0.001152841723524034, -0.7407393455505371, 0.9519354104995728, -0.7939069271087646, 1.8555656671524048, 0.3523547351360321, 0.9000807404518127, 0.386249303817749, 1.267653226852417, 1.0113266706466675, -0.27411991357803345, 0.6930524706840515, -0.5014510750770569, 0.3916241228580475, 1.5870026350021362, -0.44250455498695374, 0.391735315322876, 0.22912931442260742, 0.11667598783969879, -0.11069577932357788, -1.446917176246643, -0.8393306732177734, 2.2987072467803955, -0.843437671661377, -0.812301754951477, 0.3421359360218048, -0.4279865324497223, 0.6335196495056152, 0.36500635743141174, 0.40494003891944885, -0.3175486922264099, -1.150636076927185, -1.938451886177063, 0.6264628171920776, 1.8171167373657227, -0.29485195875167847], [1.204623818397522, -0.24990029633045197, -0.5132026076316833, 0.9397397041320801, -0.5190287828445435, -0.5433672070503235, 0.16350460052490234, -1.3162214756011963, 0.13842763006687164, 0.9985970258712769, 0.788216769695282, -0.7948733568191528, 0.3547355532646179, -0.6002185344696045, -0.44716575741767883, 0.17729288339614868, -2.3851757049560547, 1.1390607357025146, -0.0020744067151099443, 0.7342290282249451, 0.18857502937316895, -2.0535831451416016, -0.7543984055519104, 0.26902472972869873, 1.3822675943374634, 2.1783103942871094, 0.7610716819763184, -0.6860564947128296, 0.484953373670578, -0.07837475836277008, -0.3528870940208435, -1.7116003036499023, -1.876184105873108, -0.01465670671314001, -0.7662127017974854, 0.629934549331665, -0.3764960765838623, -0.016866378486156464, -0.9481991529464722, 0.4721103608608246, 0.7587487697601318, -0.2675940692424774, 0.5525479912757874, 1.2812471389770508, 0.4024743437767029, 0.23060345649719238, -0.4074086844921112, -0.8415455222129822, 0.627531886100769, 1.010528802871704, -1.483922004699707, 0.3856896460056305, -1.9397987127304077, -0.3014877736568451, 0.11016708612442017, 2.5171706676483154, -0.9946481585502625, 2.511519432067871, 1.041925072669983, 1.2131373882293701, -1.1222491264343262, 0.5082563161849976, -1.1616333723068237, -1.9609342813491821], [-0.029835298657417297, -0.20003819465637207, 0.2217741161584854, 1.2925926446914673, -1.1639972925186157, -1.5362775325775146, -1.5038959980010986, 1.4268150329589844, -0.21333865821361542, -0.1436261683702469, -0.4288250207901001, -0.3544084131717682, 0.3697930872440338, 0.4776431918144226, 0.11945830285549164, -1.775795817375183, -0.7262810468673706, -0.7584150433540344, -0.28774985671043396, 1.1055721044540405, 0.692064106464386, -0.4545322060585022, -0.2563590705394745, -1.6455961465835571, -1.3049064874649048, 1.3776386976242065, 0.8522669672966003, -0.23677855730056763, -0.576156497001648, -0.07005466520786285, -0.7431904077529907, -1.1540031433105469, 0.1252385675907135, 0.39832720160484314, -0.3769570291042328, 1.439832091331482, -1.091983437538147, 1.0204768180847168, -1.0090827941894531, 1.572601079940796, -1.4723998308181763, -2.7925665378570557, -0.6807505488395691, -0.384430468082428, 1.437174916267395, 1.0880872011184692, -0.14760294556617737, -0.3935681879520416, -0.9723068475723267, -0.17433398962020874, 0.8296834230422974, 1.7743433713912964, 0.43543919920921326, -1.5632349252700806, -0.25939637422561646, 2.337982416152954, 1.824950098991394, 0.7402545809745789, 0.0828711986541748, 1.6260535717010498, 0.7874492406845093, 0.9889923334121704, 2.1283376216888428, 0.6068584322929382], [-0.5410440564155579, 1.3016897439956665, 0.12264492362737656, -0.20942018926143646, 0.2744395434856415, -1.1011857986450195, -1.837182641029358, 0.469072550535202, 0.47311079502105713, -0.8976234197616577, -0.3596894443035126, 0.7933656573295593, -0.18979045748710632, -0.48866006731987, -0.4513469338417053, -1.7232894897460938, -0.07289639115333557, -1.4112324714660645, -1.238910436630249, -0.17128561437129974, -0.5256179571151733, 0.9249986410140991, -0.7580119967460632, -1.4011263847351074, 0.1793886125087738, -0.3730529546737671, -0.5415898561477661, 0.2577141225337982, 1.6714640855789185, 1.4446656703948975, 1.907560110092163, 1.0800151824951172, -0.7608991265296936, 0.7005243897438049, 0.21701982617378235, -0.39719682931900024, -0.8841542601585388, -0.5408271551132202, 0.19691592454910278, -0.5345175266265869, -0.37685444951057434, 0.8443898558616638, 1.3155142068862915, 0.3253975212574005, -0.4354467988014221, 1.1757497787475586, -1.0375622510910034, -0.996540904045105, -1.8454781770706177, -0.8750537037849426, 0.4110090136528015, -1.091481328010559, -1.7165969610214233, 0.44829288125038147, -1.3349789381027222, -3.2674312591552734, 0.6882860660552979, 1.090484619140625, -0.32383736968040466, -1.028928279876709, -0.3828248381614685, -1.463560700416565, 0.7536970376968384, 0.04055599495768547], [-1.661399245262146, 0.03227793797850609, -1.0193760395050049, 0.040446892380714417, -0.570305347442627, -1.104773759841919, -0.36267852783203125, 1.3358044624328613, -0.8979368209838867, -1.9792420864105225, 0.6644238233566284, 0.46891847252845764, 0.0277134720236063, 0.5113871097564697, -1.3475022315979004, -0.44727763533592224, 0.7591261267662048, -1.1767395734786987, 0.40284743905067444, 0.23293080925941467, -0.2539113163948059, 0.02129998430609703, 1.2842772006988525, 0.24942190945148468, 0.8738423585891724, 0.9256271123886108, -1.2148106098175049, 0.09033796936273575, -1.6538617610931396, 1.2739671468734741, 0.9946122169494629, 0.8087806105613708, 1.0192869901657104, 0.8509238362312317, -0.454465389251709, -0.47333812713623047, 0.880603015422821, 0.5249532461166382, 0.331440269947052, 0.5380389094352722, 1.821968674659729, 0.7881903052330017, -1.0149871110916138, 0.46508315205574036, 1.0768481492996216, -0.8900042772293091, -2.1510121822357178, 2.790156364440918, 1.9878907203674316, -3.021177291870117, -1.1754578351974487, -1.4129630327224731, -1.335498332977295, -0.4319894015789032, 0.9611131548881531, 0.07985368371009827, -0.6557276248931885, -1.0626354217529297, 0.02004326693713665, 1.740412712097168, 1.8730008602142334, 0.20722486078739166, -0.8359470367431641, 0.23224805295467377], [0.5251664519309998, 0.09025928378105164, 2.3097424507141113, -0.13671259582042694, -0.6589441895484924, -0.784996747970581, -0.8949071764945984, 1.792405366897583, 0.30044984817504883, 0.7786369919776917, -1.0666120052337646, -0.2485329806804657, -1.1248512268066406, -0.8093729019165039, -0.38274431228637695, 0.5361990332603455, 1.032010555267334, -0.28031125664711, 1.1676756143569946, -0.1606086641550064, 0.030629891902208328, -0.11062799394130707, 1.1258816719055176, -1.009026050567627, 0.3998384177684784, -0.1582650989294052, -1.2451112270355225, -0.006261318456381559, -0.19074124097824097, -0.601773202419281, 1.114759087562561, -0.45263969898223877, -0.8473204970359802, 1.3200470209121704, -0.18088608980178833, -0.7758683562278748, 1.2445673942565918, -0.613153338432312, 0.9896857142448425, 0.8521550893783569, 0.8984819650650024, -0.300240159034729, 1.2004337310791016, 0.6618061661720276, 1.2235873937606812, 0.03249626234173775, 0.4583127200603485, 0.9171299338340759, -1.0417343378067017, 0.11999528110027313, -1.1519933938980103, -1.1256707906723022, -0.17193900048732758, -0.363454133272171, -0.031001970171928406, -0.4858301877975464, -0.7283123731613159, -0.4323139786720276, -0.013784999959170818, -0.18064658343791962, -1.0236245393753052, 0.15008249878883362, 0.17378173768520355, -0.4638289511203766], [0.9125618934631348, -0.23416674137115479, -0.21891163289546967, -0.11597301065921783, 2.385873317718506, 1.1168442964553833, -0.14225099980831146, 2.253735303878784, 0.9389410018920898, -0.1634591668844223, -1.5980843305587769, -0.5909850001335144, 0.9222814440727234, -0.618723452091217, 1.2476118803024292, 0.1151951476931572, -0.6685547232627869, -1.1461677551269531, 0.10900875926017761, -0.7908079028129578, -0.18878090381622314, -0.1920221894979477, -0.5611708164215088, -0.4649636149406433, -0.3176233172416687, 1.0152852535247803, -0.32725414633750916, 1.4340486526489258, 0.2436143159866333, 0.9417625665664673, -0.14243891835212708, -0.5425183176994324, -0.18690361082553864, -0.820232093334198, 1.8991464376449585, 1.5109654664993286, 1.5466231107711792, -0.4098077416419983, -0.8716021180152893, 0.15186065435409546, -0.4699948728084564, 0.8002009391784668, -0.5968831777572632, -0.061617087572813034, -0.23445361852645874, 0.9506785273551941, 1.018571376800537, 0.781201183795929, -1.2195565700531006, -0.45707017183303833, -0.3174538314342499, 2.0731494426727295, 0.09931321442127228, -0.4056679308414459, 0.9824268221855164, -0.7874475717544556, 0.23561187088489532, -0.32809144258499146, 0.40918391942977905, -1.2465578317642212, 0.33212724328041077, -0.4826626479625702, -0.3206499516963959, -0.6003472805023193], [0.24203656613826752, -0.2912772595882416, -0.13271024823188782, -0.9099090695381165, 1.6718913316726685, -0.7079047560691833, 1.5958061218261719, 0.7524877190589905, 0.6100393533706665, 2.095020055770874, -0.7631176114082336, -0.42262160778045654, -1.4632205963134766, -0.05625385046005249, -0.8237460851669312, -0.5639963746070862, 1.631902813911438, -0.49700891971588135, -0.3879166543483734, 0.5154153108596802, -0.37290096282958984, -0.5525978803634644, 0.5612038969993591, -1.202552318572998, -1.5048999786376953, -0.5156913995742798, 0.6387205123901367, -1.407561182975769, -0.6561412811279297, 0.22969838976860046, 0.6678948402404785, -0.7488875389099121, 0.5212708115577698, -0.661757230758667, -0.9745112061500549, -1.5286563634872437, -0.6009125709533691, -1.0614094734191895, 0.20470936596393585, 1.5358930826187134, 1.7225360870361328, -1.576072096824646, 0.5286346077919006, -0.09215329587459564, -0.16508115828037262, 0.714250922203064, -1.251724362373352, -0.06465298682451248, -0.4764242470264435, 2.018778085708618, 0.2318495661020279, -1.6577736139297485, 0.7102546095848083, 0.5660727024078369, 1.6163207292556763, -0.7102009057998657, 0.2802075147628784, 0.5574367046356201, -0.7864266037940979, 0.7278173565864563, -0.18832814693450928, -0.37463921308517456, 0.08348910510540009, -1.3054558038711548], [-1.1101617813110352, 1.0755287408828735, 1.8045603036880493, -1.3164443969726562, 0.9402625560760498, -1.2583820819854736, 0.06522087752819061, 0.35213184356689453, -0.3391597867012024, -0.9158684015274048, -0.2932184040546417, -0.24376294016838074, -0.4629661440849304, 0.6116859316825867, 1.2492091655731201, 0.7827233076095581, -0.5499243140220642, -0.19373564422130585, -0.4198874235153198, -0.89956134557724, 1.6240965127944946, -0.025116151198744774, 1.4542996883392334, 2.1340205669403076, 0.6541118621826172, -0.17166094481945038, 0.5665009617805481, -0.4488973915576935, 1.949928641319275, 0.04029485583305359, 0.8286519646644592, 0.7695297002792358, -1.4518964290618896, 0.3206078112125397, 0.8008856177330017, 0.9593398571014404, 0.36534324288368225, -1.164936900138855, -0.3291865885257721, -3.197423219680786, -0.2320658266544342, 2.2650856971740723, 0.9422969818115234, -0.49375659227371216, -0.6869304180145264, -0.8773172497749329, 0.2169688642024994, 1.0270837545394897, -0.3808877170085907, -0.8029876351356506, -1.3987107276916504, 0.9560277462005615, -0.49281632900238037, 1.068955421447754, -0.4223101735115051, -0.47083526849746704, -0.7526910901069641, -1.8331537246704102, 0.40309491753578186, -0.6088058352470398, -0.18223664164543152, -0.7600995898246765, 0.5587124824523926, 0.3032960891723633], [-1.0383342504501343, 0.8840446472167969, -0.036010220646858215, 0.060486823320388794, -0.3945499062538147, -0.853824257850647, 0.586032509803772, 1.3381537199020386, 1.2949104309082031, -1.2927123308181763, -1.2223209142684937, -1.9115371704101562, 1.6846803426742554, -1.3102892637252808, -2.0776479244232178, -1.065614938735962, -2.0361664295196533, 0.6864210367202759, -1.206438660621643, -0.537556529045105, 2.047919750213623, 0.2579856216907501, -3.085240125656128, -1.0736562013626099, -0.8900302052497864, 0.5082488656044006, 0.8965453505516052, -0.6760367751121521, -0.22955012321472168, 0.7796162962913513, -0.5143357515335083, 0.0853152871131897, -1.0777053833007812, -0.6010659337043762, 2.107304334640503, -0.842481255531311, -0.37157386541366577, 1.1820036172866821, 0.5380580425262451, 2.512566089630127, -0.44575396180152893, -0.07685523480176926, -1.7542997598648071, -0.49269381165504456, 2.425630807876587, 1.56044340133667, 2.4655511379241943, 0.36211228370666504, -0.12316560745239258, -0.36903250217437744, -0.6259045004844666, -1.5859583616256714, -0.6858001351356506, -0.12654174864292145, 0.8872596621513367, 1.9922236204147339, -1.2082403898239136, 0.7830051779747009, -0.3292309641838074, -1.4272003173828125, 0.5945072174072266, 0.6398154497146606, 0.8709239959716797, 0.6931217312812805], [0.5811781287193298, -0.12116538733243942, -0.33998769521713257, 1.1449294090270996, 0.5518524050712585, 0.7108882069587708, 0.11297550052404404, 1.509390115737915, -0.09137379378080368, 0.16065320372581482, -0.4977042078971863, -0.6367749571800232, 0.6124299764633179, -0.69181889295578, -1.4685150384902954, 0.4762907028198242, -0.008643433451652527, -0.47354230284690857, 0.30289945006370544, 0.3072645366191864, 0.31184741854667664, -0.9001712203025818, -0.5702089667320251, -0.24940162897109985, 0.338527649641037, -1.4853304624557495, 0.9272831678390503, 0.6175644993782043, 0.3579038381576538, -0.4515831172466278, 0.7205889225006104, -0.0064501212909817696, -0.3577738404273987, 0.37365075945854187, 0.43679699301719666, -1.2984638214111328, 0.2008354365825653, -0.700792133808136, 0.4211263954639435, -1.104469656944275, -1.2295193672180176, -1.3334729671478271, 1.5318955183029175, -1.7714449167251587, -0.6169958710670471, -1.6936862468719482, 0.4895785450935364, -1.2013856172561646, -1.1877278089523315, 0.7728338241577148, -0.5493716597557068, -0.5474534034729004, 0.7207757234573364, 0.9862257242202759, -1.0448704957962036, 0.37389180064201355, -0.19962982833385468, -0.6765841841697693, 0.8643040657043457, -1.4737040996551514, 0.180297389626503, 0.9144929051399231, -1.661658525466919, 0.993249237537384], [0.9144186973571777, 0.409511536359787, -1.9776448011398315, 0.9656352400779724, 1.0590490102767944, -1.1480506658554077, -0.25536420941352844, -0.4991474151611328, 1.4384779930114746, -2.5492496490478516, 1.4681322574615479, 1.102806806564331, -0.2469189614057541, -0.84064120054245, 0.2189914584159851, -0.12116004526615143, -0.3734744191169739, 1.2951364517211914, 0.8915730714797974, 1.1515253782272339, 0.8231952786445618, 0.5840669870376587, -0.6061139702796936, 0.44258350133895874, 0.01489738654345274, 2.2701878547668457, -0.32659634947776794, -0.2133370339870453, 0.6288662552833557, -0.2874818444252014, 1.0134341716766357, 1.1194497346878052, 0.12739261984825134, 0.019131792709231377, -0.22954915463924408, -0.40084323287010193, 0.6225606203079224, 0.4002251923084259, -0.6863791346549988, 1.1992899179458618, -0.6990585327148438, -0.4190099537372589, -0.8630900382995605, -0.4719081223011017, 1.2195816040039062, 0.7825882434844971, -0.0962417721748352, -1.9127752780914307, -0.5481735467910767, 0.3401852250099182, 0.4785555601119995, -2.1978275775909424, 0.15032248198986053, -1.1915655136108398, 1.0704525709152222, 0.20936380326747894, -1.5925403833389282, -0.08311943709850311, 0.9971930384635925, -2.220226526260376, -0.5127075910568237, -0.013376223854720592, -0.044285740703344345, 0.6052053570747375], [0.9316248297691345, 1.0057882070541382, 2.9158778190612793, 0.3404509723186493, -0.6450645327568054, 0.13306273519992828, 0.9504312872886658, -0.3324379324913025, -0.15832079946994781, -1.7763636112213135, -0.20021581649780273, 1.2009303569793701, 0.8686173558235168, 1.0562548637390137, 1.8271287679672241, 0.2409796267747879, -1.3685497045516968, -0.589876115322113, -0.8322100639343262, 0.5412876605987549, -0.5584167838096619, -0.46176210045814514, -0.5199220180511475, 0.07705796509981155, -0.7374114394187927, -0.1845770627260208, 0.6301808953285217, 0.10941923409700394, -1.6164836883544922, -0.05487874522805214, 1.2491459846496582, -0.30492469668388367, -0.2058643251657486, -1.8800870180130005, 1.1515597105026245, -0.7221347093582153, 1.0790268182754517, -0.18383102118968964, -0.5766919255256653, -0.8061294555664062, -0.20045050978660583, 0.5348878502845764, -1.4817719459533691, -0.32050880789756775, 0.8347111344337463, -0.37367066740989685, -0.19534577429294586, -0.30092552304267883, -0.25133436918258667, -1.1324318647384644, -0.14778128266334534, 0.9987969994544983, -1.4194755554199219, 2.381544828414917, 0.8379960656166077, -0.11814092844724655, 0.7770281434059143, 0.5474526882171631, -0.010607785545289516, 0.3962072730064392, 0.5601469278335571, 1.8499586582183838, 1.0601052045822144, 0.8245868682861328], [0.18441849946975708, -0.3131469488143921, -0.10798955708742142, 0.8544285893440247, -1.5976868867874146, 1.0454474687576294, 0.4017762839794159, -0.21143972873687744, 1.0165694952011108, -0.13172554969787598, -0.5323254466056824, 0.30853453278541565, -0.4623299539089203, 1.5026715993881226, -1.2204989194869995, -0.33802416920661926, -0.4709620177745819, -0.7168704271316528, -0.4045206308364868, -2.160841703414917, -0.4874936640262604, 2.3718173503875732, -0.8177666664123535, 1.239770531654358, -1.2247693538665771, -1.6455484628677368, 0.8016343712806702, -1.9247629642486572, -0.26272594928741455, 0.2770603895187378, -0.9848967790603638, 1.056522250175476, 0.6868582367897034, 0.3008703887462616, 1.3024752140045166, 1.0628514289855957, 0.6736735105514526, -0.7600495219230652, -0.06567877531051636, -0.6397550106048584, -2.4223954677581787, 2.236696720123291, -1.3779653310775757, 1.5709037780761719, 0.8744381666183472, 0.8817350268363953, -0.1786702573299408, -0.66612309217453, 0.10916406661272049, -1.0534688234329224, -0.04489050433039665, 1.521683692932129, 0.395288348197937, -1.240609049797058, 0.22065232694149017, -1.3033411502838135, 0.32726937532424927, -0.45626965165138245, 0.03696993738412857, -1.8439778089523315, -0.8637722134590149, 0.4615117311477661, -1.0852041244506836, -1.3907183408737183], [-0.8531891703605652, -0.8788555860519409, -0.20846857130527496, -0.23294945061206818, 0.4213595986366272, -0.6575110554695129, -1.0438590049743652, 0.7124961614608765, 0.4457707405090332, 0.2843839228153229, 1.8615365028381348, 0.166897252202034, -0.3861250877380371, -0.6397836804389954, 0.34746813774108887, 0.4461624324321747, -0.7026980519294739, -0.338613361120224, 0.15293839573860168, 1.2664713859558105, 0.059297893196344376, -1.3859670162200928, -0.549704909324646, -1.4708465337753296, -0.19093528389930725, -0.4036587178707123, -0.13583621382713318, 0.7086982727050781, -0.4292715787887573, 0.11624275892972946, -0.6893202066421509, 0.49024122953414917, 1.9290521144866943, 1.8058537244796753, -1.1618942022323608, 1.1962611675262451, 0.5822452306747437, 2.1645684242248535, 0.40454867482185364, -1.0515632629394531, -0.7261607646942139, -0.6620243787765503, -0.26273131370544434, -1.0753822326660156, -0.4252443015575409, -0.11705829203128815, -0.7733404040336609, -1.4481111764907837, -0.682613730430603, -0.9207446575164795, 1.3405873775482178, 0.5229364037513733, -1.151065707206726, -0.00819437112659216, 0.3936767876148224, 0.27680155634880066, 0.27625468373298645, 0.710588812828064, 0.17151691019535065, -1.1988834142684937, -0.7912791967391968, -0.40921148657798767, 0.6520221829414368, -0.22841273248195648], [-0.13858599960803986, 0.20815445482730865, 0.20288844406604767, 0.3993369936943054, -0.5482693314552307, 0.03493509814143181, 0.25769487023353577, -0.1303066611289978, 0.07976092398166656, -0.12416285276412964, 0.6450074315071106, 0.38866737484931946, -1.0295507907867432, -0.16031880676746368, -0.24224169552326202, 0.43160220980644226, -0.2678838074207306, -0.6213324666023254, 0.3530489504337311, 0.056906379759311676, 0.8436574339866638, -0.6099944710731506, -1.3964416980743408, -2.0264716148376465, 0.20234648883342743, -1.1672334671020508, -0.27507349848747253, 0.38909801840782166, 0.15758928656578064, -0.6381781697273254, -0.9362547993659973, 0.5713061094284058, 0.4680916965007782, 1.055863857269287, -0.13357718288898468, 0.20091751217842102, 0.4901941120624542, 1.0072931051254272, 0.050710540264844894, 1.8307955265045166, 0.1737125962972641, -0.21719828248023987, -0.20839016139507294, -1.3841636180877686, 1.434653878211975, 1.447984218597412, -0.9744012355804443, 1.4536906480789185, -0.2011757493019104, -1.2874783277511597, -1.618841528892517, -0.4132808446884155, -0.4020792841911316, -0.5652344226837158, -0.6868815422058105, -0.5420117974281311, -0.40912142395973206, 0.24243168532848358, -0.9227097034454346, -0.3228141665458679, -1.7046154737472534, 1.101482629776001, -0.2261657565832138, -0.30921489000320435], [0.5886233448982239, -0.7989237308502197, 0.37482884526252747, -0.07630787789821625, 1.8377752304077148, 0.2415417581796646, -0.3550766706466675, -0.24435453116893768, -1.7286454439163208, 2.9015164375305176, 0.12486990541219711, -0.1771988570690155, -0.8942481279373169, 0.4052905738353729, 0.2295648157596588, -0.5132634043693542, 0.13812346756458282, -0.3527545630931854, -0.09048672765493393, 0.31094175577163696, 0.08824162930250168, 0.8246771097183228, -0.37742748856544495, 0.7928188443183899, -0.20965956151485443, -1.1233104467391968, 1.1359361410140991, -0.40621399879455566, 0.12998992204666138, 0.2483295351266861, 0.7662431597709656, -0.055982187390327454, 0.7450763583183289, -1.6917476654052734, 1.0220654010772705, -0.44159048795700073, 0.01835772581398487, 0.8394890427589417, -1.85585618019104, -1.1743066310882568, 1.2134770154953003, 0.39141613245010376, 0.8843033313751221, 0.9228286743164062, -0.6844433546066284, 0.8382556438446045, -1.093171238899231, 0.6435971260070801, -0.24331237375736237, -0.12818388640880585, -0.2937633991241455, 0.48312073945999146, 0.041918594390153885, 1.121375560760498, 1.3641003370285034, 0.2500375509262085, -0.6022546887397766, 2.6892130374908447, 2.6091227531433105, 1.4829868078231812, 0.9327779412269592, 1.4604798555374146, 0.080167755484581, 0.70055091381073], [-0.8741149306297302, -1.2423523664474487, -0.014113969169557095, 0.7927822470664978, 0.7727216482162476, 0.6628018617630005, -0.0910845622420311, 0.4290519654750824, 0.022354111075401306, 0.2654290795326233, -0.2509424090385437, 0.0029100237879902124, -0.28197288513183594, -0.9891958832740784, 1.1645587682724, 1.509636640548706, 1.8835241794586182, -1.0190235376358032, 0.04362878575921059, 0.021400632336735725, 0.43756037950515747, -0.31350675225257874, -1.3860197067260742, 0.12161029130220413, -0.5202368497848511, -1.1720737218856812, -0.5388950109481812, -0.16414126753807068, 0.2512098550796509, -0.6618878245353699, 0.784990668296814, -1.708183765411377, -0.2774176001548767, -1.1554840803146362, -0.12092790007591248, 0.06616753339767456, 1.4153056144714355, -0.19742678105831146, 0.05933988466858864, -1.7762248516082764, 1.3265036344528198, 1.1177424192428589, -0.15043531358242035, 1.833373785018921, -1.0938429832458496, -0.4373902976512909, -1.4389872550964355, 0.0729164332151413, 1.156264066696167, 1.0021052360534668, 1.1303786039352417, 0.9226210117340088, -0.6012166738510132, -0.39058488607406616, -0.7522111535072327, -0.646210253238678, -1.201682448387146, 1.0487529039382935, -0.4836808443069458, 0.7112727165222168, 1.4610992670059204, -0.2774108648300171, 0.848540723323822, 0.9131507277488708], [-0.5594849586486816, -0.24787382781505585, -1.1757464408874512, 1.7817420959472656, 1.7238078117370605, 0.07451160252094269, -0.7586610913276672, -0.551202118396759, -1.5833702087402344, -0.021942278370261192, -1.1696099042892456, -1.0165174007415771, -0.06791581958532333, 1.2314624786376953, 0.35487663745880127, -0.7565487027168274, 1.9020406007766724, 0.4390394389629364, 0.9696183800697327, 1.0615540742874146, -1.0513784885406494, -0.22992604970932007, 1.793521761894226, 0.2820863425731659, 0.8042874932289124, -0.7187666893005371, -0.11181633174419403, -0.5473441481590271, 0.33826136589050293, 0.3142956793308258, -0.36783847212791443, 0.1334165334701538, 0.8284748792648315, -0.19607770442962646, 0.43490487337112427, 0.9103896021842957, -0.30607083439826965, -1.1217907667160034, 0.9637740254402161, 2.19907283782959, 0.43545177578926086, -0.359596848487854, 0.2739261984825134, -2.015230178833008, -0.34184983372688293, 0.3214990794658661, 0.8660144209861755, -1.5837886333465576, -1.4641634225845337, 1.310389518737793, 1.0798194408416748, 0.4811806082725525, 0.6898097991943359, 1.4353642463684082, -0.3431673049926758, 0.6341519355773926, 0.8259078860282898, 1.0169565677642822, 0.4126073718070984, 1.4598848819732666, 0.952003002166748, -0.7945813536643982, -0.8803603053092957, -0.2654556930065155], [0.19933047890663147, 0.9464505314826965, 0.32716986536979675, -2.551259994506836, 1.9300886392593384, -0.711037278175354, 0.004205118399113417, -2.406041145324707, 1.0513454675674438, 1.3270642757415771, 0.5409901142120361, -0.39601466059684753, 0.11384371668100357, 1.8999797105789185, -0.53206467628479, -0.859420895576477, 0.06884469091892242, -0.4786319434642792, 0.4991527199745178, -2.262463331222534, -0.6644049882888794, 1.559767484664917, 0.7067237496376038, 1.3901079893112183, -0.3242948055267334, 0.7471365928649902, -2.60986065864563, 0.4845174551010132, -0.5646410584449768, -0.0026550914626568556, 0.9689953923225403, -0.4461330473423004, 0.18935759365558624, 0.2616463899612427, 0.15102995932102203, 0.13876710832118988, 1.8134377002716064, -1.2569611072540283, 1.0637543201446533, 0.08037129044532776, 0.15594729781150818, -0.2827288508415222, 0.5962924957275391, -0.906923770904541, 0.48354023694992065, 0.8513494729995728, -1.3357161283493042, -0.30337706208229065, 1.0310806035995483, 0.7504709362983704, 0.0842253714799881, 0.5877379179000854, -0.26478925347328186, -0.5551730990409851, -1.8455311059951782, 0.2818453311920166, 0.03653861954808235, 0.7088054418563843, 0.14991404116153717, 0.08979208767414093, 1.0725144147872925, -1.2413848638534546, 0.5160589218139648, -1.0069471597671509], [0.580400288105011, -0.7920028567314148, 0.667216420173645, 0.16714483499526978, 0.30088162422180176, 0.4403863251209259, -0.6741024851799011, -0.3153105080127716, -0.16042926907539368, 1.1479324102401733, 0.3429337441921234, -1.2778899669647217, 1.427141785621643, -0.18409433960914612, 1.2655627727508545, 0.059947118163108826, 0.36551356315612793, 1.1264642477035522, 2.116678237915039, -0.6606360673904419, 0.9992788434028625, 2.1858367919921875, -1.1151363849639893, -0.6954604387283325, -1.0686614513397217, -0.5530418753623962, 0.7463154196739197, -0.5735496878623962, -1.9449447393417358, 0.025984464213252068, 1.22359299659729, -1.5987122058868408, 0.8553666472434998, 0.736350417137146, -0.5237066149711609, -0.29347437620162964, 2.014296293258667, 0.36736971139907837, -1.3559000492095947, -0.1976555436849594, -0.688721239566803, -0.5982056856155396, -0.8018976449966431, -0.4883279800415039, -0.46533313393592834, 0.07722480595111847, -0.8033127784729004, -1.6345713138580322, -0.6378790140151978, -0.2303425520658493, 0.045746397227048874, -0.18072821199893951, 0.43059197068214417, 0.07791728526353836, -1.2595123052597046, -1.0215474367141724, -0.42516613006591797, 1.6212663650512695, -0.3313254415988922, -0.885280191898346, 0.6044290065765381, -0.1483500897884369, 0.3716890811920166, 0.49222299456596375], [-0.15744651854038239, 0.17204086482524872, -1.4243626594543457, -0.39714956283569336, 1.0849179029464722, -1.5946929454803467, -0.46373793482780457, 0.7488438487052917, 0.4692612290382385, -1.3654276132583618, -1.7897394895553589, 0.4975270926952362, 0.4943447411060333, -0.31263431906700134, 1.32307767868042, -0.2903774082660675, -0.9836138486862183, 1.2344028949737549, 0.9933933019638062, 0.24466295540332794, 0.040866341441869736, 0.2637424170970917, -2.139474630355835, 0.6632161736488342, 0.3179500699043274, -0.7638258934020996, 1.0041749477386475, -0.012689848430454731, 0.9537270665168762, 0.8980122208595276, -0.2480442374944687, -0.5988965034484863, 0.2795964479446411, 1.213640809059143, -1.5441030263900757, 0.06471279263496399, 0.2746315002441406, -1.870772123336792, 1.8611820936203003, -0.9683079719543457, 1.0008959770202637, 0.23488080501556396, 0.28462859988212585, 1.3047711849212646, -0.26380228996276855, 2.2661056518554688, 0.4660395681858063, -1.720889687538147, -1.3438255786895752, -0.9296898245811462, 0.7677770256996155, -0.1649046093225479, -0.1352159082889557, 2.971187114715576, -1.7828892469406128, -2.3496174812316895, -0.8969369530677795, 2.709075450897217, 0.19601362943649292, -1.0466668605804443, 0.527831494808197, 1.0819021463394165, -0.20282503962516785, -0.47237902879714966], [-0.46040070056915283, 0.9777954816818237, -0.11931386590003967, 0.8773460984230042, 0.07166696339845657, -1.7586075067520142, -0.8053646087646484, -0.854403018951416, -1.2114617824554443, 0.683678388595581, 0.6211780309677124, 0.14323748648166656, -0.7180012464523315, 0.13867205381393433, -0.11053327471017838, 0.565963625907898, -0.09045807272195816, 0.4895855188369751, 0.3147056996822357, 1.0951262712478638, -0.9123926162719727, 0.3209744095802307, -0.8921770453453064, -0.927342414855957, 0.6213961243629456, -0.0178761538118124, 0.6176143884658813, 0.9235701560974121, -2.167476177215576, 0.9136724472045898, 0.7672975063323975, -1.1054290533065796, -0.2099536806344986, 0.48333704471588135, 0.8972133994102478, 0.06601466238498688, 0.5289061665534973, -0.1706792265176773, 0.3350861966609955, -0.3142493665218353, -0.3984310030937195, -1.3959784507751465, 0.9880074858665466, 1.9754211902618408, 0.8633911609649658, -1.6747684478759766, 0.4155435264110565, -1.0458109378814697, -1.208590030670166, -0.9414167404174805, 0.3470759689807892, 2.128760576248169, -0.6195902824401855, -1.4569402933120728, 0.32248446345329285, 0.13357791304588318, 0.23149988055229187, -0.23326139152050018, 0.9571572542190552, -0.14422011375427246, -0.283833771944046, -1.9789611101150513, -0.3094383776187897, -0.27300965785980225], [0.0812024399638176, 0.8823996186256409, 0.13582965731620789, -2.242258310317993, -2.059937000274658, 0.673759937286377, -0.9077293276786804, -0.47405895590782166, -0.12480760365724564, 0.089272640645504, -0.5660448670387268, -0.17711296677589417, 0.1993512511253357, 0.8668360710144043, 0.5274118781089783, -0.14799179136753082, -0.2375420331954956, 0.17619691789150238, -1.019173502922058, -0.33675140142440796, -0.3393041789531708, -0.6058424115180969, -0.08660893887281418, -0.6346612572669983, 0.9599772691726685, 0.1122613400220871, 0.4709872603416443, 2.1070029735565186, 0.010153498500585556, -0.834854245185852, 1.0983160734176636, -0.6956260800361633, 0.4182102084159851, -0.5272231698036194, 1.558438777923584, -0.7731626033782959, -0.258466899394989, 0.8300668597221375, -0.3528618812561035, -0.9444526433944702, -0.4804325997829437, 0.3828882873058319, 0.7159587740898132, 0.13755130767822266, 0.23564371466636658, 0.6358039379119873, -0.6666387915611267, 0.9431325197219849, -0.5962768197059631, -1.6588222980499268, -0.28887853026390076, -0.638837993144989, 0.3645007610321045, 0.458907812833786, 1.8355716466903687, 0.6556342840194702, 1.6343954801559448, 2.620748519897461, 0.06035292148590088, 0.949069082736969, -1.1164634227752686, 2.0128836631774902, -1.7501226663589478, -0.5739983320236206], [-1.312820315361023, -1.686279058456421, 0.2344556301832199, -0.8671186566352844, 1.797980546951294, 1.0184251070022583, -1.6559489965438843, -0.5777209401130676, 0.6635436415672302, -0.2875919044017792, -0.3013427257537842, 0.7132595181465149, -0.8809494972229004, -1.988886833190918, -0.7868351340293884, -2.010262966156006, -1.1401218175888062, -0.09778626263141632, 0.25687986612319946, -0.9829402565956116, 1.3534259796142578, 0.529716968536377, -0.7794011831283569, -1.3997657299041748, 0.38283464312553406, 1.1461764574050903, 0.2579432427883148, 0.11596544086933136, 1.0263005495071411, 0.9513402581214905, 1.0460894107818604, -0.7687765955924988, -0.3032398521900177, 0.8357940316200256, 0.7360212206840515, 0.7387682795524597, 0.7156460881233215, -0.7538806200027466, -0.0453900508582592, -0.6642351746559143, -0.6500027179718018, -0.6822222471237183, -1.4112330675125122, -0.21054545044898987, -0.5260010361671448, 1.5670280456542969, 0.4444611072540283, -1.4256482124328613, 0.21387487649917603, 0.3459315299987793, 0.05782483518123627, 1.3788551092147827, -0.31812331080436707, 0.05645507946610451, 1.4823391437530518, -0.537725567817688, -1.1112477779388428, 0.3499803841114044, -0.8999171853065491, -1.579723596572876, 1.2451947927474976, 0.3853326141834259, -1.3190046548843384, -0.824198305606842], [-0.5341749787330627, -0.9999505877494812, 0.6254498362541199, -1.2567301988601685, -1.122046709060669, 0.009378906339406967, 0.7878450751304626, 3.2825396060943604, -0.11670531332492828, -2.700612783432007, -0.22386787831783295, 1.3479549884796143, 1.0888311862945557, 0.36873823404312134, 1.5954651832580566, 1.3480569124221802, -0.7580240964889526, -0.215918630361557, -0.08148246258497238, 0.4223296642303467, 0.5808550119400024, 0.25493085384368896, -0.8185977339744568, -0.3198215961456299, -1.483744502067566, 1.1784571409225464, -0.7030459642410278, 0.07712041586637497, 0.27937087416648865, 0.8361589312553406, 1.3332502841949463, 0.3853529095649719, -0.7749696373939514, -0.43589431047439575, 0.6184977889060974, -0.36187484860420227, 0.8521261811256409, 0.6072938442230225, 0.6987863183021545, 0.4073452055454254, -0.1480453759431839, 0.9213875532150269, 1.9304612874984741, -0.22329290211200714, 0.127630352973938, 0.860592782497406, 0.18784315884113312, -0.6713162660598755, -0.927738606929779, 1.1337648630142212, 0.8911373615264893, 1.558409333229065, 1.5712578296661377, -1.0720242261886597, -0.758192777633667, 1.6310263872146606, -0.8102781772613525, 0.607400119304657, -1.1809064149856567, 1.1351507902145386, 0.7314459681510925, 0.12365119159221649, 0.7954714894294739, -0.20916134119033813], [1.1206077337265015, 0.5886415839195251, -2.169090509414673, -0.08790173381567001, -0.4135516285896301, 0.31395503878593445, -0.24314278364181519, -1.440608024597168, 0.9142798185348511, 0.007852144539356232, -0.2019854038953781, 0.5157341957092285, -0.05683545023202896, 0.7772846221923828, -0.32274413108825684, 0.3371034860610962, 0.15971636772155762, 1.1906465291976929, 0.919121265411377, -0.47723063826560974, -0.014357890002429485, 1.4989070892333984, 0.8029950261116028, 1.8962912559509277, 0.7196400761604309, 0.9825453758239746, -0.30036604404449463, 0.7548298239707947, -2.2415964603424072, -0.3752228021621704, -0.2116311937570572, 1.4118444919586182, 0.7543107867240906, -0.45630091428756714, -0.5816650986671448, -0.20380376279354095, 0.7692894339561462, 0.7511840462684631, 0.40150612592697144, -0.5278804898262024, 0.6855437159538269, -0.921229898929596, -0.7095507979393005, -1.1707559823989868, -0.17930907011032104, 0.6989723443984985, 0.026888394728302956, 1.4941966533660889, 0.08154843002557755, 0.2302708625793457, -0.013341418467462063, -0.6097922921180725, -1.524206519126892, 1.231067419052124, -0.7364915013313293, 1.4452482461929321, -0.8324069380760193, -0.8258392214775085, 2.5400278568267822, 1.5266392230987549, 1.5006953477859497, -0.6437927484512329, -1.727483868598938, 0.24706661701202393], [0.04384884983301163, -1.2565762996673584, -0.4660770297050476, 0.28690841794013977, -0.5107067227363586, 0.015021879225969315, -0.7636079788208008, 0.44835174083709717, -0.8973925113677979, -1.6051504611968994, -0.12020626664161682, 0.3035884201526642, 0.6519176363945007, 0.6247398853302002, -1.1338502168655396, -1.2373371124267578, 0.6676783561706543, -0.3221237063407898, -1.0930789709091187, 0.642312228679657, 0.6854863166809082, -1.450379490852356, 1.7042467594146729, 1.2913212776184082, 0.8248606324195862, 0.657241940498352, 0.2430962324142456, -0.228378564119339, -0.8930649161338806, -0.9179550409317017, 0.4232414662837982, 2.2637979984283447, 1.343268871307373, -0.41930529475212097, 0.7177938222885132, 1.8396289348602295, 0.881706714630127, 0.44135135412216187, 1.8260077238082886, -0.08499445021152496, -0.2628181278705597, 0.4045897126197815, 1.8561416864395142, 0.25443851947784424, -0.26316559314727783, 2.354029417037964, -0.24020999670028687, 0.7635588049888611, -1.3650728464126587, -0.4986569285392761, 0.2035616785287857, -0.3938322961330414, -0.16887058317661285, -0.3431532382965088, -2.027935266494751, 0.09434075653553009, -0.9257756471633911, 0.9227330088615417, 1.9049880504608154, 0.08552394062280655, 1.9288389682769775, 0.45817863941192627, -0.6876744031906128, -1.0289537906646729], [-0.3785306513309479, -0.5964143872261047, -1.9147844314575195, -1.1693363189697266, -0.18729889392852783, 0.6303409934043884, 0.9863729476928711, -0.3141252398490906, 0.3669412136077881, 0.14151963591575623, -1.195134162902832, 0.2702319622039795, 0.06981302797794342, -0.8021724820137024, -0.13440802693367004, -1.0013707876205444, 0.81312495470047, 1.9816938638687134, -0.21127937734127045, 1.1720610857009888, 0.4208189845085144, -0.16035781800746918, 0.8206865787506104, 0.6114331483840942, 0.7396600842475891, -0.6488701701164246, -1.9741727113723755, -0.6690479516983032, -0.7007823586463928, 0.8268990516662598, 0.7915893793106079, -1.1724485158920288, 0.857388973236084, 0.8302123546600342, -1.1609137058258057, -0.870465874671936, 1.6943873167037964, -0.9399497509002686, 0.15366177260875702, 0.6046481728553772, 1.2751110792160034, -0.03617629408836365, -0.4500961899757385, 0.6104171276092529, 0.6839226484298706, 0.19729113578796387, -1.3243948221206665, -0.39950284361839294, -1.4201594591140747, -0.6973406672477722, -0.2363337278366089, 1.0137526988983154, 0.3079782724380493, 0.4961957335472107, 0.37757259607315063, -0.5871328115463257, 0.8468865156173706, 1.6427900791168213, -0.009652312844991684, -0.9864449501037598, 0.9113908410072327, 1.3174479007720947, -0.12247008830308914, -1.6926788091659546], [0.543878436088562, -1.6578404903411865, -1.084505319595337, -1.3153973817825317, 0.3819590210914612, 0.20659036934375763, -0.4387679696083069, 0.9316399097442627, -0.6845875382423401, -2.0785224437713623, -0.6622335314750671, -0.3232342302799225, -0.879223108291626, -0.64077228307724, -0.09215408563613892, 0.38517317175865173, 0.10252702236175537, -0.5439684391021729, -0.40148258209228516, 0.10491765290498734, 0.4051486551761627, 1.0275626182556152, 0.06685622781515121, 2.0125880241394043, 0.07732567936182022, 0.6528477668762207, -0.7863008975982666, 0.3264354467391968, 0.7746242880821228, 1.4270960092544556, 0.2883283793926239, 0.07638402283191681, -0.44464564323425293, 1.21699857711792, 2.890023708343506, -1.1398718357086182, 0.2656748592853546, 0.587973415851593, 0.5785131454467773, -0.5692815184593201, 1.5158793926239014, 0.3419129550457001, -1.215287208557129, 0.26554253697395325, 2.05946946144104, 0.5168347358703613, -2.254535436630249, 0.7344229221343994, -0.6338294148445129, 1.1643646955490112, 0.003043669741600752, -0.3832522928714752, -1.0881699323654175, 0.7770740389823914, 0.41712531447410583, -1.7179656028747559, 1.0551538467407227, -1.545501708984375, -0.5773816704750061, -0.3426383435726166, 0.3873879313468933, -0.006140603683888912, 0.24249547719955444, -0.20279155671596527], [1.1314319372177124, 0.6729838848114014, 0.4214443564414978, -0.11332479119300842, -2.2333335876464844, -0.6855576038360596, 0.7705621123313904, 1.1572413444519043, 1.466997742652893, 0.16082817316055298, 0.10796379297971725, -1.4900026321411133, -0.6589651107788086, 0.9423607587814331, 0.3836934566497803, -0.08393647521734238, -0.24934686720371246, 0.4441007971763611, 0.5849788188934326, 0.5843559503555298, -0.8610277771949768, -0.9777320623397827, -1.7317872047424316, -1.4387418031692505, 0.7472278475761414, 0.9462347626686096, -0.4640766382217407, 0.6915079951286316, 1.0223900079727173, 1.4794896841049194, 1.5349655151367188, 0.7038491368293762, -0.8070816993713379, 0.08461810648441315, 0.3601989150047302, -0.3902727961540222, -0.8004055619239807, 0.3919145166873932, 0.8911592960357666, 1.2313683032989502, 0.5479575991630554, -1.4460258483886719, 1.8260780572891235, 0.9179889559745789, -0.8737738132476807, 1.612613558769226, -0.9068034291267395, 1.4914686679840088, -0.31312569975852966, 0.993029773235321, 0.1568583995103836, -0.5684024691581726, -1.1622307300567627, 0.046330153942108154, -0.688549816608429, 0.991273045539856, -1.0176701545715332, 0.5010995864868164, -0.2186318039894104, -0.2694917321205139, 0.5214523673057556, -0.6597891449928284, 0.4297420084476471, 0.3154534697532654], [0.7650720477104187, 0.4192257225513458, 0.5941323041915894, -0.6026943325996399, -0.8860428333282471, 1.8923232555389404, 0.6793313026428223, 1.039132833480835, 1.2357434034347534, -0.727418065071106, -0.1158057376742363, 1.245105504989624, 0.13261418044567108, 1.4725749492645264, -0.8824074268341064, 0.3329308032989502, -0.28351670503616333, 0.054685357958078384, 1.0470112562179565, -1.1660314798355103, 1.4609886407852173, -0.10804228484630585, -1.4834412336349487, -0.47403770685195923, -0.029248125851154327, -0.414119154214859, 0.4606182873249054, 1.086674690246582, -0.1656138151884079, -0.2428857833147049, 0.5974977612495422, 0.9490812420845032, 0.4435465633869171, 1.4134050607681274, 1.4035919904708862, -0.7342958450317383, -1.0189621448516846, 1.4635945558547974, 0.7245184779167175, -0.093169204890728, -0.33524423837661743, -0.2557508647441864, -0.353819340467453, 1.4451380968093872, 2.0871975421905518, 0.4005773067474365, -0.19381068646907806, -0.6270812749862671, -0.8711786270141602, 0.3465932309627533, 1.6830627918243408, 1.314028263092041, 1.0104615688323975, -0.5134865641593933, -0.15294523537158966, 0.01119782030582428, -0.12637466192245483, -0.23029477894306183, -0.3211139440536499, -0.5326395034790039, -0.9246577024459839, 0.7591793537139893, -0.03847847878932953, -2.3579342365264893], [1.3975034952163696, -0.07312226295471191, 1.01986563205719, -0.9307202100753784, 0.5064679384231567, -0.8072706460952759, 0.6444306969642639, 1.011432409286499, -0.26743239164352417, 0.4745532274246216, 0.5546784996986389, 0.20263154804706573, 0.255351185798645, -0.5660692453384399, 0.9701902866363525, 1.8057204484939575, -0.1884370595216751, -0.1636001467704773, -0.08004647493362427, 0.29147136211395264, 0.07883752137422562, -0.09706658869981766, 0.1655251830816269, -0.5201013684272766, 0.2810359001159668, -0.11610355228185654, 0.5030754804611206, 1.0441553592681885, 0.09165151417255402, 1.0705132484436035, 0.536196768283844, 0.708340048789978, -1.301222562789917, -1.04219388961792, -1.2710702419281006, 0.10720670223236084, -1.3831746578216553, -1.4391528367996216, -1.8636630773544312, 1.8270518779754639, -0.30904534459114075, 0.14067764580249786, -1.445742130279541, 1.4857592582702637, -1.0181165933609009, 0.3280782699584961, 2.0833537578582764, 0.6064198613166809, -0.6619279384613037, -1.2016855478286743, -0.6220189332962036, -0.3719865679740906, -1.2265030145645142, -1.6566015481948853, 2.2796123027801514, -0.965793788433075, -0.7187343835830688, 0.8248680233955383, -0.8639259934425354, 1.824005365371704, -0.8115220665931702, 0.728787362575531, 2.667757749557495, -2.137333393096924], [-0.3098682761192322, 1.9227110147476196, -0.1622622311115265, -0.3833170533180237, 0.9108201265335083, -0.04783002659678459, -1.0691602230072021, -0.04281599819660187, -0.695496141910553, 0.09319429099559784, 0.3592051565647125, 1.763429880142212, 0.287971556186676, -0.3940353989601135, -0.8474740386009216, -0.44207465648651123, 0.32166460156440735, -0.3115871250629425, -0.007319317199289799, 1.1611814498901367, 1.1027164459228516, -0.08329879492521286, -1.805887222290039, 1.6079168319702148, 1.9118696451187134, 0.12787745893001556, -1.6336586475372314, -0.7660502195358276, 1.0423153638839722, 1.1921498775482178, 1.1656122207641602, 1.2909696102142334, 0.9055080413818359, -0.5143525004386902, 0.14765752851963043, 0.9300529956817627, -1.5444029569625854, -0.27029743790626526, -0.1713278740644455, 0.45241501927375793, -2.03473162651062, 1.3002742528915405, -0.9760280251502991, 0.5244084596633911, -0.3872455358505249, 1.2720139026641846, -0.33886435627937317, -1.150721549987793, 1.0836714506149292, 1.1074503660202026, 2.3963851928710938, -0.9835947155952454, 0.9487074017524719, 0.5815428495407104, 0.28786420822143555, 0.3035438358783722, -1.575236201286316, 1.0467438697814941, -0.22871142625808716, 0.7685230374336243, 1.0723190307617188, -1.512072205543518, 1.2241286039352417, -0.3819742798805237], [0.5575888752937317, -2.139538526535034, -0.20195287466049194, -0.5239827632904053, -0.7627149820327759, 0.8747752904891968, -0.9714066982269287, -1.2980616092681885, -0.1652684062719345, -0.4355635643005371, -0.5863510370254517, -0.7778312563896179, -0.6329174041748047, -0.6589335799217224, -1.241644263267517, 0.3864445388317108, -0.38238751888275146, -0.46496906876564026, 0.18899014592170715, 0.23049437999725342, -0.38645368814468384, 0.41086921095848083, 0.7031761407852173, -0.25159314274787903, -0.34525635838508606, 1.9310487508773804, 1.2910432815551758, -2.767681121826172, -1.1602507829666138, 0.5488259196281433, -1.8656420707702637, -0.1825464963912964, -0.048554614186286926, -0.08036837726831436, -1.1969146728515625, 0.6036810278892517, 1.1727771759033203, -2.217529058456421, -0.645488977432251, 0.29552552103996277, -0.9831621646881104, -0.1097654402256012, -1.0794577598571777, 0.7698801159858704, -1.0863898992538452, -0.8346379399299622, 0.35604411363601685, -1.314725637435913, 2.301436424255371, 0.7666006088256836, -0.8457192778587341, -1.3661943674087524, 0.7959074974060059, -0.8052543997764587, 0.5961592197418213, 1.1058276891708374, -1.6868271827697754, 0.9921892881393433, 1.2715461254119873, -0.08350484073162079, 0.007471640128642321, -0.5627371072769165, -0.704053521156311, -0.45256897807121277], [0.49216827750205994, -1.0187023878097534, -1.6321226358413696, 0.8488567471504211, -0.1930537074804306, 0.9201437830924988, 0.09437599033117294, 2.0333735942840576, -0.16408111155033112, 1.319048285484314, 1.1113852262496948, -1.7027119398117065, -0.3768381178379059, -1.1266196966171265, -1.407339334487915, -1.800558090209961, -0.19338490068912506, -1.6772063970565796, 1.067753553390503, 1.1013681888580322, 1.3227840662002563, -0.028680654242634773, -1.7224652767181396, 1.557903528213501, -0.4448202848434448, -0.8567668199539185, 1.168047308921814, 0.24078260362148285, 0.8022955656051636, 0.39135801792144775, -0.09136700630187988, 0.9795907735824585, 1.0133055448532104, 0.6442470550537109, 0.5556690692901611, 0.7818520069122314, 0.6992140412330627, -0.26011914014816284, 1.231691837310791, -0.9162654876708984, -1.4588391780853271, -0.14225199818611145, -1.781790852546692, 0.4386322796344757, -0.3082202970981598, 1.101883053779602, -2.856611490249634, 0.23552533984184265, 0.031488511711359024, -0.7474055290222168, -1.01375412940979, 1.3986371755599976, -0.26303184032440186, 0.16429255902767181, -1.1772128343582153, 0.48385173082351685, -1.0469481945037842, 0.7896607518196106, -1.1183756589889526, -0.4672519266605377, 0.6715512871742249, -1.259738802909851, -0.06791263073682785, 1.2629265785217285], [0.3153538405895233, 1.0474530458450317, -0.5276336073875427, -0.5783289074897766, 0.3724975287914276, -1.1733454465866089, 1.543582558631897, -0.6778109073638916, 1.673387050628662, 0.32987022399902344, 0.8445594310760498, 0.18533466756343842, 1.4578460454940796, 0.1497420370578766, -0.2015751451253891, -1.0338484048843384, 0.8785139322280884, -0.6483086943626404, 0.0874372124671936, -1.1204017400741577, -0.6151247024536133, -0.9423622488975525, -1.6407737731933594, -0.45378807187080383, 0.5926295518875122, -1.0070405006408691, 0.723628044128418, -2.296170711517334, -0.0071289995685219765, 0.5991363525390625, 0.6969776153564453, 2.5317938327789307, -0.0804692804813385, 0.35749348998069763, 0.11519285291433334, -1.052863597869873, 0.34892651438713074, -0.9342861771583557, -0.5112893581390381, 1.2913533449172974, -0.08437652140855789, 2.121255397796631, 1.2871204614639282, -0.785240650177002, 0.8316009044647217, -0.743108332157135, 1.322070837020874, 1.1125209331512451, -0.4638645052909851, 0.9024979472160339, -0.9691933989524841, -1.4596625566482544, -1.219720721244812, 2.4675309658050537, -1.0581756830215454, 0.40376198291778564, -2.683712959289551, -1.064399242401123, -0.5650650858879089, 1.5791302919387817, -0.04077261686325073, -1.8006681203842163, -0.8744012117385864, 0.04523424804210663], [0.251811146736145, 0.9674201011657715, 0.756610095500946, 0.22251874208450317, -0.7265446186065674, -1.427971363067627, 0.40291827917099, -0.32010015845298767, 0.2711177170276642, 0.3420027494430542, -0.34749042987823486, 0.5418103337287903, -2.9313206672668457, 1.0595877170562744, -1.0513743162155151, -1.112729549407959, -1.0297584533691406, 1.6664073467254639, -0.003308933461084962, 1.5113003253936768, 1.2786941528320312, 1.1791044473648071, 1.2788385152816772, 0.6585536599159241, 0.5876768827438354, -0.3658484220504761, -0.6148249506950378, 1.0580650568008423, 2.0020198822021484, 1.3735960721969604, 0.8352101445198059, -0.002391367917880416, -0.545090913772583, -1.7593954801559448, -0.31672680377960205, -0.8420753479003906, 1.8756327629089355, 0.33584165573120117, 0.5425822138786316, 0.9357538819313049, 0.6405004858970642, -0.898759663105011, 0.5506783723831177, -0.005436995532363653, -0.847824215888977, -0.8102406859397888, -0.17437618970870972, 0.14099033176898956, 0.4712406396865845, 0.8879580497741699, -1.283004641532898, 1.1460566520690918, 0.1054437905550003, 1.8953649997711182, 2.0392072200775146, 0.1394314169883728, 0.8565036654472351, 0.5750653147697449, 0.3110238313674927, 0.03588612750172615, -0.3433426022529602, 0.26392731070518494, -0.5531227588653564, 0.6333101987838745], [1.4135056734085083, 0.1458836942911148, 1.869039535522461, -0.3437519669532776, -0.18474598228931427, 0.46848222613334656, -0.8489392995834351, 0.0070634325966238976, -0.24012695252895355, -1.019174575805664, 0.8857039213180542, -0.8328759074211121, 0.6548163890838623, 1.2318074703216553, 0.3625587821006775, -2.6071949005126953, -0.7233144044876099, -2.383373737335205, 0.5445134043693542, -0.33414506912231445, 0.2804967164993286, -1.4498893022537231, -0.1893736869096756, -1.3304637670516968, 0.2382422536611557, -0.948834240436554, -1.0065146684646606, -0.2249847650527954, -0.22626011073589325, -0.307953417301178, 0.49355900287628174, -0.017775028944015503, 0.9956035614013672, 0.2371751368045807, 0.5502627491950989, -1.2706149816513062, -0.21749171614646912, 0.0765310525894165, 0.1403178721666336, 0.09138084948062897, -0.6163246035575867, 1.2829980850219727, -0.7423391938209534, 0.37511757016181946, 0.40507709980010986, 0.18973125517368317, 0.2977209985256195, 0.1642855405807495, 0.7585611939430237, 2.9062728881835938, 0.5040161609649658, 0.45518752932548523, 2.259064197540283, 0.6616004705429077, 2.230017900466919, 0.24669799208641052, -0.9724538326263428, 0.7632764577865601, 0.3128430247306824, -0.48712146282196045, -0.3303539454936981, 0.11801980435848236, 1.14680814743042, -1.5102860927581787], [-0.8122624158859253, 2.181591749191284, -1.6513020992279053, -1.5320881605148315, -0.49317437410354614, -1.0956807136535645, 1.0448545217514038, -0.9211987853050232, -0.3735695779323578, 1.3928641080856323, -1.357654094696045, 0.3702179491519928, -2.693164348602295, -1.1031684875488281, -0.6236599683761597, -0.34130480885505676, -1.2215569019317627, -0.25100862979888916, 0.6289070844650269, 1.2591667175292969, -0.5466616153717041, -0.5691700577735901, 0.6528109908103943, -1.2097891569137573, 1.8203328847885132, -1.0354360342025757, 0.6498191356658936, -0.05560923367738724, -0.378337562084198, -1.4192787408828735, 0.5834847092628479, 1.433659315109253, -0.806423008441925, 0.8078320622444153, 0.8243227005004883, -1.4664005041122437, 2.380824565887451, 2.539109230041504, 1.9908241033554077, -0.8330209255218506, 0.3449389636516571, -1.2672922611236572, 0.5675937533378601, -0.7842413783073425, -1.0028769969940186, -0.023509617894887924, -0.43011999130249023, -0.366396427154541, 2.184605360031128, 0.6821394562721252, -0.9127985239028931, -1.0832356214523315, -0.4901379942893982, 0.6455276012420654, -1.48167884349823, 0.4561953842639923, -0.5643604397773743, 1.2531654834747314, 0.5867253541946411, -0.4821678102016449, -1.207633376121521, 0.5116010308265686, -0.43145546317100525, -0.21509720385074615], [-0.7082425355911255, -0.15973427891731262, -0.9321808815002441, 0.25921568274497986, 0.4234023988246918, -0.8738941550254822, 0.5508305430412292, -2.0839805603027344, -1.5839718580245972, -0.19234952330589294, -0.5152528882026672, -1.2545534372329712, -1.0314480066299438, -0.12523174285888672, -1.158466100692749, -0.9155590534210205, -0.16041411459445953, 1.9293524026870728, -1.3394371271133423, 0.32053548097610474, -1.5208890438079834, 1.9635207653045654, 0.42652520537376404, 0.06270421296358109, -0.7197691202163696, 0.05849926546216011, 1.7224478721618652, 2.4685556888580322, 0.6007898449897766, -0.6397508382797241, -1.0408353805541992, -0.26470789313316345, -0.6969926953315735, -0.09288107603788376, -0.9996681809425354, 0.2355349361896515, -0.3748825192451477, -1.3499771356582642, -1.7936166524887085, 1.3509060144424438, 0.4651680588722229, 0.3246895372867584, 2.574707508087158, -0.3007352650165558, 1.8707386255264282, -1.1443047523498535, 0.7301696538925171, -0.023272031918168068, -1.0274732112884521, 0.7483125329017639, 0.08640272915363312, 1.8697279691696167, 0.14890159666538239, -0.420320987701416, -0.039791662245988846, 0.4302945137023926, 0.46835240721702576, 0.01460985653102398, -0.3463468849658966, 0.32849377393722534, 0.7709019184112549, 0.07362177968025208, -0.7910025715827942, -0.010261298157274723], [-1.2433912754058838, -0.8992546796798706, -0.1715538650751114, 0.9631408452987671, -0.24160656332969666, 1.3323564529418945, -1.6604392528533936, -0.4026896059513092, -0.6708723902702332, 1.071943998336792, -0.19791144132614136, 1.2716395854949951, 0.4876629114151001, 0.05604417994618416, -1.4721993207931519, -1.4835271835327148, -0.02296164631843567, -0.44442540407180786, -0.1391783356666565, 0.04632514715194702, -1.7350497245788574, 1.6512707471847534, 0.34059372544288635, 1.1653308868408203, -0.8009776473045349, -1.2526044845581055, -1.1814459562301636, -0.6686230897903442, 0.7814923524856567, 0.6412168741226196, 1.3790972232818604, 0.6849550008773804, 0.8456576466560364, 1.399609088897705, 0.21469947695732117, -0.2268172800540924, 0.09837549924850464, -0.3347502052783966, -0.12183104455471039, -0.24065415561199188, 0.1608899086713791, 0.120479516685009, -0.14383141696453094, 1.3304533958435059, 0.4298301041126251, 1.8348129987716675, 0.4019843637943268, -1.6503980159759521, 1.3327500820159912, 1.4507579803466797, 0.9045165181159973, 1.2962439060211182, 0.08249682188034058, -0.606241762638092, -0.5722146034240723, 1.768515944480896, 0.1443084180355072, 0.3813089430332184, -0.12039604783058167, -0.24560467898845673, 2.706873655319214, -0.77720707654953, 0.04167779162526131, 1.1123969554901123], [0.21061496436595917, -0.6947557926177979, 0.3395612835884094, 0.706581175327301, 0.6405547261238098, -1.2828625440597534, 0.5582827925682068, 0.5082600712776184, 0.10218524932861328, 0.02310054376721382, -0.22496485710144043, -0.3829715847969055, 1.6038647890090942, 0.3723568320274353, -0.24059545993804932, -0.12880346179008484, -1.175896406173706, 0.33862510323524475, -1.161431908607483, -1.4801678657531738, 0.10325219482183456, -0.737096905708313, 0.2791346311569214, -2.2979471683502197, -0.46875664591789246, 1.9865144491195679, -0.35844600200653076, -0.5089349746704102, 0.9252551794052124, 0.14746086299419403, 0.19062314927577972, -0.32299795746803284, -0.4184110462665558, -1.2551909685134888, 2.2938857078552246, 1.6235836744308472, 0.3408648371696472, -0.6223602294921875, -0.7077314257621765, -0.7637214660644531, -0.22679482400417328, 0.6703956723213196, 0.6891216039657593, 0.282117635011673, -1.589621901512146, -0.37097012996673584, 0.6996645331382751, -1.501116156578064, 0.08312811702489853, -0.8179647326469421, -0.45402321219444275, -0.8639034628868103, -0.24406473338603973, 0.7857351899147034, 1.6256427764892578, -0.8436945676803589, -0.4393658936023712, 0.17018622159957886, 1.380067229270935, -1.616563320159912, 0.2752097249031067, -0.9185234904289246, -0.11438214778900146, 1.728890061378479], [0.9262374639511108, 0.7223699688911438, -0.4401733875274658, 1.818045735359192, 1.9451192617416382, -1.517004132270813, -0.5057533979415894, 0.6563842296600342, -0.11306288093328476, -0.8595195412635803, -0.5604023337364197, 0.5601698756217957, -0.649276077747345, 0.7749934792518616, -2.4637372493743896, -2.370943307876587, 0.20810313522815704, 1.069610595703125, 0.9105636477470398, 0.0036561053711920977, -0.14975984394550323, -1.8057886362075806, -0.3109385669231415, -0.48385652899742126, 1.3652973175048828, -1.0313373804092407, -0.7394313216209412, -0.6701041460037231, -1.898047685623169, -1.9971096515655518, 1.1911969184875488, -0.1474149525165558, 1.0490803718566895, -2.729616641998291, 0.07499511539936066, 1.179254412651062, -0.8889368772506714, -0.06422434747219086, 0.9288576245307922, 1.121456265449524, -1.270951509475708, -0.5621118545532227, 2.117859363555908, -0.531964898109436, -0.5021063685417175, -0.5074071288108826, -0.10203699767589569, 0.8011234998703003, -1.571531891822815, 2.6130423545837402, -0.4516910910606384, 0.27619996666908264, -0.19954556226730347, 0.25644034147262573, 0.07711292803287506, -0.5359945893287659, -0.6196883320808411, 0.6627836227416992, 0.3077106177806854, -0.9157797694206238, -2.0603702068328857, -0.3602582812309265, 0.16853521764278412, 0.9589832425117493], [-0.7534578442573547, -0.3205021023750305, 0.6036524176597595, -0.09015888720750809, -0.15146297216415405, 1.634610652923584, 0.38423317670822144, -1.4474109411239624, 1.7335952520370483, -1.109392762184143, -0.5570315718650818, -0.40305742621421814, 0.9441964626312256, 0.1509527564048767, 1.1727149486541748, 0.11472135782241821, 0.26656728982925415, 0.6508673429489136, 0.8189384341239929, 0.05904676765203476, -0.3532123267650604, 1.3025319576263428, 0.48000696301460266, 1.2534164190292358, -0.4066060185432434, -2.292527675628662, 3.4123148918151855, -0.5583013296127319, 0.467116117477417, -0.7105022072792053, 0.4050848186016083, 1.5961827039718628, -0.6145563125610352, -0.5485668182373047, 0.577490508556366, -0.07829637080430984, 1.947551965713501, -0.3187420070171356, -1.841946005821228, -1.740655779838562, -0.25618281960487366, 0.21735893189907074, -0.16565954685211182, -0.4230492115020752, 1.1312036514282227, 0.23397667706012726, 0.8375938534736633, -0.6694262027740479, 1.0131498575210571, -1.0061945915222168, 1.034403681755066, -0.014186306856572628, -1.02220618724823, 0.6661491990089417, 0.31882065534591675, -1.4225693941116333, -0.8161489963531494, 0.8550350666046143, 0.2481137067079544, -0.44381311535835266, -0.37268975377082825, -0.8356979489326477, 0.9542925953865051, -0.9692131876945496], [-1.7837365865707397, 0.012564164586365223, -1.4011090993881226, 1.9296029806137085, 0.7842428088188171, 0.7326080203056335, 0.16440856456756592, 0.9981335401535034, -0.22294588387012482, -0.585253894329071, 0.7378033995628357, -0.37277770042419434, -0.14598752558231354, 0.040648847818374634, 1.7848223447799683, 0.1276993751525879, 2.1725387573242188, 0.11298418790102005, -0.7694764137268066, 1.4284452199935913, 0.7284892201423645, -1.000430703163147, 0.21415509283542633, 1.6511765718460083, -0.06316062062978745, -0.4668548107147217, 0.8676909804344177, -0.17610470950603485, -0.724906325340271, 0.6124312877655029, 1.0574380159378052, -0.11007994413375854, 0.5679050087928772, -0.32723814249038696, -2.2202858924865723, -0.4578266739845276, -0.4098179340362549, 1.6938201189041138, 0.07805684953927994, 1.1267637014389038, 0.4863007664680481, 1.1656309366226196, -0.598197877407074, -0.9826244115829468, 0.47747546434402466, -1.1524090766906738, 1.4773372411727905, -0.9715239405632019, 1.1975016593933105, -0.29211845993995667, 0.5494993925094604, 0.5964291095733643, -1.5949310064315796, 0.24564731121063232, -2.0719919204711914, 1.5025724172592163, -0.29571741819381714, -0.294609010219574, -0.1537065953016281, 1.0784810781478882, -0.30810272693634033, -0.4631156027317047, -0.568890392780304, 0.7187281847000122], [-0.8739988207817078, -0.33857259154319763, 0.14552117884159088, 0.5219683051109314, 0.4717884361743927, -0.9554004669189453, -1.1026523113250732, -0.28686341643333435, 0.3489055931568146, 2.337656021118164, 1.134084939956665, 0.33816587924957275, -0.7440569400787354, -0.8406355977058411, -0.6055103540420532, -0.5988406538963318, -1.1113266944885254, -0.30827099084854126, 0.03276896849274635, 0.7058015465736389, -0.12436941266059875, -0.8895174860954285, 0.8573065996170044, -1.844753384590149, 0.1651361584663391, 0.3079700767993927, 0.10408613830804825, -0.304453581571579, 0.03622988983988762, -0.06794402003288269, -0.8665658831596375, -0.40954893827438354, 0.4643568694591522, 0.7419325113296509, -1.316200613975525, -0.49000993371009827, 0.8929257392883301, -0.2011866271495819, -0.19269467890262604, -0.38501906394958496, 1.1069142818450928, 1.4654242992401123, 2.109503984451294, -1.237072467803955, -1.2616711854934692, -0.04045455530285835, 1.3066080808639526, 0.20528779923915863, -2.042231321334839, -2.3298888206481934, -0.14964671432971954, -1.5814828872680664, -1.102810025215149, 0.034758683294057846, 0.17216040194034576, -1.9631810188293457, -0.7527238726615906, 0.9660242795944214, -0.08131100982427597, -0.15635044872760773, 0.24351337552070618, 0.5878354907035828, 1.1991325616836548, -0.8013158440589905], [0.8551665544509888, 1.288844347000122, 0.8808268308639526, 1.3648914098739624, 0.2509821653366089, -2.2676236629486084, -0.5768359303474426, 0.7996662259101868, -0.20387019217014313, -2.565109968185425, -1.1863129138946533, -1.293123483657837, -0.6279308199882507, 0.5298404097557068, 0.6883419752120972, 0.16428892314434052, 0.4857388436794281, 1.4456284046173096, 0.31804540753364563, 0.020219890400767326, -1.6431524753570557, 1.4726741313934326, 0.3179585039615631, 1.0876110792160034, -1.4422872066497803, -0.7969668507575989, 0.10851453244686127, -1.5005797147750854, 2.2411820888519287, 0.35943639278411865, 0.03498987853527069, -0.844014048576355, 1.1206212043762207, -0.1817377656698227, 0.48157748579978943, 0.19959211349487305, -1.0350803136825562, -1.3236829042434692, 0.3145037591457367, 0.3621257543563843, -0.0048388405703008175, 0.8669946789741516, 1.4996259212493896, 2.616436004638672, -0.5955671072006226, -1.0606399774551392, -0.05703836306929588, 1.0776275396347046, 1.915046215057373, 1.4821559190750122, 1.7825539112091064, 1.4871907234191895, -0.10596339404582977, -0.6277347207069397, -0.6066933274269104, 0.0767076388001442, -0.8908308744430542, -2.0666725635528564, 0.4289485216140747, 0.4693131744861603, 0.04781190678477287, -0.8090876340866089, -1.0256669521331787, -0.8328477740287781], [1.829779863357544, 0.6079836487770081, -0.032740771770477295, 0.7535216808319092, 0.3094562590122223, -0.32960352301597595, 0.5671166777610779, 0.7889261841773987, 0.32924219965934753, -0.14299850165843964, 2.1978299617767334, -1.6192384958267212, 0.2028515636920929, 0.6765373945236206, -0.23415039479732513, 1.1046661138534546, 1.0450295209884644, -1.0522360801696777, 0.7555248737335205, -2.5486574172973633, -0.2238229364156723, -0.7876675724983215, -0.09565544128417969, -0.31497684121131897, -1.3981351852416992, -0.40891173481941223, -1.428792119026184, 0.07782641798257828, -0.3909763693809509, 0.4836280941963196, -1.7094846963882446, 0.8222242593765259, 0.5572905540466309, -1.2238727807998657, -0.9273296594619751, 1.213436484336853, -0.7695770263671875, 0.7406057715415955, 0.07075107842683792, 0.9336544871330261, -0.4000643491744995, -1.5400851964950562, -0.06487516313791275, 0.09499675035476685, 0.8410851359367371, -0.9657964110374451, -0.4272550046443939, -0.2620998024940491, -0.24089060723781586, 0.16194382309913635, 0.14686934649944305, 1.0283180475234985, -0.6503424644470215, 0.7672204375267029, 0.317352294921875, 0.6443566679954529, -0.7428578734397888, 1.0735360383987427, 0.16545945405960083, -1.2743511199951172, 2.1066994667053223, 0.11098508536815643, -0.08973588049411774, 0.5230947136878967], [-1.8852583169937134, 0.5014067888259888, -1.4705277681350708, -1.6466598510742188, -0.19768008589744568, -0.9217662811279297, -0.11761130392551422, 0.5392274856567383, -0.07589038461446762, 0.0611959807574749, 0.3695867955684662, -0.44342052936553955, 0.6854764819145203, -0.07543441653251648, -0.4606954753398895, 0.5785716772079468, -0.34616824984550476, 1.0439555644989014, -0.06614848226308823, 0.5109833478927612, 0.7785614728927612, 0.1482572853565216, -0.6807369589805603, 1.0089467763900757, -0.2781570255756378, -1.893239140510559, 1.0544509887695312, 0.23793378472328186, 2.5107123851776123, 0.40407633781433105, 0.44720080494880676, -1.3384144306182861, -1.1902388334274292, -0.3478032648563385, -1.3274526596069336, 0.5170037746429443, -1.2648600339889526, 0.6555992960929871, 0.4790867865085602, 0.5153640508651733, 0.8239319324493408, -0.002514923457056284, -0.726477324962616, 0.1367531269788742, -0.4389498829841614, -0.8250018954277039, -0.7848206162452698, -0.13858433067798615, -0.37159958481788635, -0.3224887549877167, -1.1855424642562866, 1.4569801092147827, -0.5123635530471802, -0.6593229174613953, 1.1770968437194824, 1.3356291055679321, -0.13149110972881317, -1.001888394355774, 0.50467848777771, 0.7120872139930725, 1.6182509660720825, 1.4606330394744873, -1.9086567163467407, -1.6975359916687012], [0.6257149577140808, -0.30011388659477234, 0.09558514505624771, -2.4125466346740723, 0.30610692501068115, 0.5213682055473328, -1.1767386198043823, -1.1837475299835205, -1.1434094905853271, 1.0736702680587769, -0.09044773876667023, 1.2265454530715942, 0.8975168466567993, 0.2703746557235718, -0.259101539850235, 0.838130533695221, -0.7341996431350708, 0.1710178256034851, -1.793238639831543, -0.21957507729530334, 1.0416204929351807, 0.6286717653274536, -2.5641496181488037, -0.5905472040176392, 0.7531139850616455, 0.500639796257019, -0.4859657883644104, -0.10119839757680893, -0.05754700303077698, -1.4327495098114014, -0.33420372009277344, 1.7420905828475952, 0.08840809762477875, -1.4677430391311646, -0.05589261278510094, 0.24510115385055542, -0.48462870717048645, 1.350901484489441, 0.2678707242012024, 2.3338282108306885, -0.20066413283348083, -1.6388866901397705, -1.5455524921417236, 0.8368446826934814, -0.29482218623161316, 1.2557481527328491, -0.5115089416503906, 0.6073752045631409, -1.6824647188186646, 0.04552830010652542, -0.3905823230743408, -1.3231197595596313, -1.1207143068313599, 0.2698637545108795, 1.0851243734359741, -0.9088281393051147, -0.43221741914749146, -0.46741369366645813, 0.19095568358898163, -0.42124149203300476, -0.44718676805496216, 2.1691715717315674, -0.4261890947818756, 0.9437812566757202], [-0.0017870372394099832, -0.06667893379926682, -0.36871469020843506, -0.8584339618682861, -0.5365592241287231, 0.8579835891723633, 1.3687776327133179, -0.28845617175102234, 2.3491687774658203, -0.31972137093544006, 0.22365286946296692, 0.4004463851451874, 1.112519383430481, 0.7998204231262207, 1.0111109018325806, 0.9062211513519287, -1.1875746250152588, -0.8004676103591919, 1.3736907243728638, 1.6460645198822021, 0.6431160569190979, 0.6061097383499146, 1.483250379562378, -0.9633086919784546, 1.0484330654144287, -1.1250061988830566, -1.4945939779281616, 0.1449354887008667, -1.1363805532455444, 2.1792898178100586, -0.0766635462641716, -1.6557108163833618, -0.7281584739685059, -3.4414639472961426, 1.4539581537246704, 2.9262077808380127, 0.6324422955513, -0.7552076578140259, -0.6641901731491089, -0.2984228730201721, -0.23394672572612762, 1.507651925086975, -0.06311062723398209, -0.6896935105323792, 1.4988235235214233, 1.003490924835205, 0.7700827121734619, 0.24390164017677307, 1.4233994483947754, 0.07319692522287369, 1.5206384658813477, -1.195302963256836, 0.5131784081459045, -0.6374373435974121, 0.2548367381095886, 0.014178337529301643, 1.623605489730835, 0.3145686388015747, -0.09766427427530289, 0.7265487313270569, 0.6556288003921509, 0.8622586131095886, 0.2448776811361313, -0.13587668538093567], [-1.8017867803573608, -0.7253479361534119, -0.9627877473831177, -1.318171739578247, -0.4357829988002777, -1.1462295055389404, -0.622043251991272, 0.3023635149002075, 0.41070711612701416, -1.6785986423492432, -0.2436593770980835, 0.7063234448432922, 0.7115907073020935, -0.7950965166091919, 0.10685839504003525, 2.5808446407318115, -0.2132285088300705, -1.1054891347885132, -0.27656811475753784, 1.633559226989746, -0.4290984570980072, 0.1031927689909935, -0.5948722958564758, -0.11505745351314545, 0.4767809212207794, 0.07318462431430817, -1.1536632776260376, 1.6074756383895874, -1.3089890480041504, -0.04628922417759895, -0.6502938270568848, -0.687781810760498, -0.6558584570884705, -0.816937267780304, -0.7998380661010742, 0.043124422430992126, -1.6289684772491455, 1.4838167428970337, -1.2324421405792236, -1.5642437934875488, 1.1628215312957764, 1.908496618270874, -0.3676907420158386, 0.17043741047382355, 1.8721630573272705, -1.8755667209625244, 1.2257028818130493, -0.8710408210754395, 0.5059248805046082, -0.4624626636505127, 0.3645603656768799, -3.0218749046325684, 0.4080997407436371, -0.2143469601869583, -0.9412243962287903, -0.7308567762374878, -0.6508904099464417, -0.3115435838699341, 0.5373802781105042, 1.4210126399993896, 1.6361032724380493, 0.8409722447395325, 0.13889148831367493, -0.3929096460342407], [-0.8445446491241455, 0.25849980115890503, -0.09692439436912537, 0.22963394224643707, 0.5007781982421875, 0.8148468136787415, 1.069831132888794, -0.16599276661872864, -1.415533423423767, -0.027845539152622223, -0.7207481265068054, 0.6766245365142822, -1.0876286029815674, 0.6696146130561829, -0.6386269927024841, 0.44247502088546753, -0.046061381697654724, 1.1488382816314697, -0.19218957424163818, -1.3090494871139526, -0.6049568057060242, -0.8576660752296448, 1.1114767789840698, -0.7345368266105652, 1.2542186975479126, -0.10233145952224731, -1.1761105060577393, -0.27566009759902954, 0.31377869844436646, -0.5684148073196411, -1.0199203491210938, -0.6673725247383118, 1.7926373481750488, 1.437034249305725, -0.09430185705423355, 1.3634587526321411, -0.9397156834602356, -0.25491559505462646, 0.6220956444740295, 0.028160080313682556, 0.7658818364143372, 0.8775128126144409, 0.16713374853134155, 0.8309894800186157, 1.5348867177963257, -1.3896235227584839, 0.02753535471856594, 1.2020095586776733, 0.5282038450241089, -0.7299059629440308, -0.9867939352989197, 0.4592365324497223, -0.4985537827014923, -0.03630993515253067, -0.034801363945007324, 0.5050481557846069, -0.3688690960407257, 0.734605610370636, -2.7353663444519043, 0.4034865200519562, -0.7177714705467224, -0.5045934915542603, -0.3721678555011749, 0.3438989818096161], [0.9405683875083923, 0.2961772084236145, 0.7114695906639099, -0.15536625683307648, -1.8793190717697144, 0.618698239326477, 0.1765572428703308, -0.31166931986808777, -0.6094690561294556, 0.012126131914556026, 0.5217315554618835, -1.4811545610427856, -0.3629440665245056, 0.466666579246521, 1.4246156215667725, 0.44918256998062134, -0.12531371414661407, -1.9468588829040527, -0.7145498991012573, -0.7058157324790955, -1.4115769863128662, 1.699196219444275, 0.08454254269599915, -3.156083106994629, -0.9014462232589722, -0.6397214531898499, -0.7194791436195374, -0.6340668201446533, -0.2547973394393921, 0.9238991737365723, 0.09784388542175293, -0.03962801769375801, 0.48051828145980835, -0.330258846282959, -0.6991368532180786, 0.38807055354118347, -1.3715946674346924, -0.3536718189716339, -0.5597518682479858, 0.9780803918838501, -0.9793884754180908, 0.205191969871521, 0.783358633518219, -0.5864206552505493, -0.45468178391456604, -0.6738813519477844, -1.0442173480987549, -1.1078599691390991, -1.2262767553329468, 1.696439266204834, 0.9988014698028564, 1.3766565322875977, 1.22294282913208, 1.614320158958435, 1.3888227939605713, 1.3191829919815063, 0.0514310859143734, -1.0662689208984375, -0.41341859102249146, -2.700493812561035, -0.974496603012085, -0.9339970946311951, -0.19144262373447418, 1.053716778755188], [0.2615833878517151, 1.030211091041565, -1.5348278284072876, 0.321473091840744, 1.2854944467544556, 0.7071738243103027, -1.2424432039260864, 0.5714728832244873, 0.26807478070259094, -1.3548922538757324, 1.958693504333496, -1.8723762035369873, -0.5800838470458984, 0.6226443648338318, -1.3915151357650757, 0.5983673334121704, -0.44047608971595764, -0.12278740853071213, -0.9397981762886047, -1.070652723312378, 0.285940945148468, -0.953457772731781, -0.5117756724357605, -0.2706562876701355, -0.9590490460395813, 0.12438155710697174, 1.313231348991394, -2.0931787490844727, 1.4761337041854858, -1.0901325941085815, 0.40886613726615906, 0.5621970891952515, -1.0203964710235596, -0.635917067527771, -1.256908893585205, -0.9342884421348572, -0.20643067359924316, -1.270870566368103, -2.234783172607422, -0.14614690840244293, 0.2713358402252197, -0.9136135578155518, 1.7996660470962524, -1.2264550924301147, 0.09708790481090546, 0.12485825270414352, -0.8088166117668152, 1.154811978340149, 0.47818079590797424, -0.23360945284366608, 0.060849033296108246, 1.0398216247558594, -0.26198697090148926, 1.139056921005249, 0.0964525118470192, 0.14262476563453674, 0.7808657884597778, -0.8153947591781616, 1.7695523500442505, -0.6893499493598938, -0.0877998098731041, -0.021174795925617218, 0.1712702214717865, 0.37405475974082947], [-0.25363391637802124, -1.5888420343399048, -1.3066812753677368, 0.1562524288892746, 0.8166002035140991, 0.3636675477027893, -1.4679170846939087, 1.5496631860733032, 0.28757983446121216, 1.2174583673477173, 0.475269615650177, -0.20658418536186218, -1.2886444330215454, 1.5574396848678589, -0.16399841010570526, 1.9884438514709473, 0.45497316122055054, -1.783534049987793, -0.37073802947998047, -1.119563341140747, -1.157696008682251, -0.3800828754901886, -0.2531278133392334, 0.6257098913192749, 1.570296049118042, -0.5560978651046753, 0.051518701016902924, 1.4644652605056763, -0.7763977646827698, 0.22645682096481323, 1.0951014757156372, 0.844167172908783, 1.501548409461975, -1.7803032398223877, -2.5484306812286377, 0.031714294105768204, -0.4266878664493561, 0.3946512043476105, 0.9135860204696655, -1.2266480922698975, 1.110804557800293, -0.7932027578353882, 0.5372549295425415, 0.5824664831161499, 0.6928030252456665, -1.4170513153076172, -2.181328058242798, 0.38713327050209045, 1.1889094114303589, -0.9883447289466858, 0.8690778017044067, 1.7995857000350952, -0.0743321105837822, 0.9742284417152405, -1.2242671251296997, -2.5297367572784424, 0.9201791286468506, 0.22032149136066437, 1.145127773284912, -0.1643494963645935, 0.430526465177536, 2.5331764221191406, -1.1560027599334717, -1.4036777019500732], [1.7288810014724731, -0.8564209342002869, 0.5819128155708313, 0.7019355297088623, 0.2649131417274475, 0.35764065384864807, -0.10181048512458801, -0.8869650363922119, 1.0195435285568237, -2.097234010696411, 1.7667580842971802, 0.30129390954971313, -0.7948404550552368, -1.8040844202041626, 0.5178060531616211, 1.1973620653152466, 0.838422954082489, 0.5236167907714844, -0.9117808938026428, -0.7067555785179138, -2.1856555938720703, 0.23987218737602234, -0.37541303038597107, -0.5807082653045654, -0.4264819622039795, -1.7806768417358398, 0.5233718156814575, 0.33448904752731323, 0.8771051168441772, -0.25109919905662537, -0.7484286427497864, -0.9039900898933411, -1.918264389038086, -1.48005211353302, 1.0539308786392212, -1.2192384004592896, 1.2443445920944214, 1.2879716157913208, 0.2471672147512436, 0.8632183074951172, -1.2406643629074097, 0.4556376338005066, -1.135318398475647, 1.1749013662338257, -0.7256648540496826, -0.5208422541618347, 0.3866037428379059, 0.7796338796615601, -0.7991134524345398, 1.8318437337875366, 1.6578351259231567, 0.8668283820152283, 1.1397902965545654, -0.40697237849235535, -2.529655694961548, -0.2108428180217743, -2.65315580368042, 1.4671107530593872, -0.9067662358283997, 1.7808071374893188, -0.4112043082714081, -0.5456045269966125, 0.9625571966171265, -0.694810688495636], [-1.0052707195281982, -1.7531306743621826, 1.4581092596054077, 0.9825966358184814, 0.6771277785301208, -0.016904449090361595, -0.07615301012992859, -1.061115026473999, 0.41824719309806824, 0.39020437002182007, -1.408835530281067, -0.3391841650009155, -1.3351517915725708, -2.079451560974121, -0.5723991394042969, 0.46431779861450195, 0.20388242602348328, -0.2671225070953369, -0.6137374639511108, -0.8009331822395325, -0.560075581073761, 0.01629868522286415, 1.3399227857589722, -1.2259700298309326, 1.580393671989441, 2.095107078552246, 0.5838435888290405, 0.4028424918651581, -1.5049242973327637, -0.0182146355509758, 0.5480146408081055, 0.7924508452415466, 1.6142354011535645, -0.2805626392364502, 1.6692198514938354, -0.42874088883399963, -0.909912645816803, 2.570651054382324, 0.2264919877052307, -0.9201053380966187, -1.3819568157196045, -0.7989873886108398, -0.26944902539253235, 0.26620230078697205, 0.758461594581604, 1.7160080671310425, -1.280943512916565, -1.6467316150665283, -0.6437814235687256, 0.6394237279891968, -0.4114968478679657, -0.807672381401062, -0.25684553384780884, 1.1455929279327393, 0.6478846073150635, 1.472367763519287, 0.465512752532959, -1.2770737409591675, -0.2802829444408417, -1.633263349533081, -1.5662531852722168, 0.09477529674768448, -1.7757186889648438, -2.2478127479553223], [0.34328514337539673, 0.11046433448791504, 0.14032214879989624, 1.0818052291870117, 0.23147691786289215, -1.0840370655059814, -0.09021808952093124, 0.5651488304138184, -1.9654853343963623, -1.0694371461868286, -1.5108728408813477, -1.1343497037887573, -1.513594627380371, 0.8894152045249939, -1.3512321710586548, -1.4218093156814575, -1.185180902481079, 0.09814770519733429, -0.13164517283439636, -0.09630691260099411, -0.34239843487739563, 0.6868821978569031, -0.7218858003616333, -0.6150830388069153, -1.58901846408844, 0.587925136089325, 0.4825383722782135, -1.1302844285964966, -1.1896166801452637, 0.4247751533985138, -0.9537158608436584, 0.01097091380506754, 1.0801169872283936, -1.2284305095672607, -0.3273056447505951, 1.1040947437286377, -0.09084044396877289, -0.1818597912788391, 0.28913745284080505, -1.3287632465362549, 0.8895276784896851, -0.06173359602689743, -0.1622820645570755, 1.3023959398269653, 0.5569954514503479, 0.8017142415046692, 0.7144762277603149, 0.06870941817760468, -0.3896732032299042, 0.4507442116737366, -0.5152081847190857, 1.2258594036102295, 0.5765032768249512, 0.161790668964386, 0.055734239518642426, 1.5696765184402466, -0.03503810241818428, 0.6915097236633301, 0.42273423075675964, 0.9831823706626892, -0.4267415404319763, -0.7377592325210571, -0.7536238431930542, 0.5965936183929443], [0.38032665848731995, -0.3968450427055359, -0.18676911294460297, 1.2645273208618164, -0.9507881999015808, 0.15907806158065796, -0.5920926332473755, -0.171543687582016, 1.0460506677627563, -0.38640397787094116, 0.31592926383018494, -1.1119894981384277, 0.24198994040489197, 0.5964126586914062, -1.975748062133789, -0.07704196870326996, 0.8704515099525452, 0.5312239527702332, 1.3282815217971802, -1.5592706203460693, 1.353764533996582, -0.4626578092575073, -0.8415949940681458, -0.7939307689666748, -0.9861096143722534, 0.7102879881858826, -0.1000543162226677, 1.4712092876434326, 1.8967355489730835, 2.4400439262390137, -1.00009286403656, -1.4399996995925903, 0.9179459810256958, 1.2608076333999634, -1.4532970190048218, -0.16564905643463135, 0.547336995601654, -1.6423009634017944, 1.2140780687332153, 2.667682647705078, 0.7611376643180847, 0.3573022782802582, 0.6713427901268005, -0.6964914202690125, 0.07836080342531204, -0.6449090838432312, -1.4891386032104492, 0.21091148257255554, 1.2513350248336792, 0.48529478907585144, -1.611352801322937, 2.297032117843628, 1.7756706476211548, -0.6591312885284424, 1.7096275091171265, -2.378439426422119, 0.7540137767791748, -0.4813706576824188, 0.20654305815696716, 0.4319290816783905, -1.409523367881775, 0.11732150614261627, -0.012097787111997604, 0.379462331533432], [-0.06500231474637985, -1.812461495399475, 0.8866446018218994, 0.719052255153656, -2.1468582153320312, 1.9042325019836426, -0.11449795961380005, 0.4617244601249695, -0.7789331078529358, 0.19425570964813232, -0.27224311232566833, -0.29998093843460083, 0.039160989224910736, 0.22982971370220184, 0.024411000311374664, -0.0407540500164032, -0.3264753818511963, -0.3695999085903168, -0.4173997640609741, 0.4733518362045288, -0.4657082259654999, 0.5388257503509521, 0.37914159893989563, -0.05814872682094574, 1.2007410526275635, -0.48556241393089294, 0.6518716812133789, 0.6134611964225769, -0.15387289226055145, 0.5740712881088257, -0.4025005102157593, 1.3232802152633667, 0.7603299021720886, -0.8136739730834961, 0.25259578227996826, -0.3031867444515228, -0.2802911698818207, -0.22783263027668, -1.2195039987564087, -1.4222197532653809, 0.5581292510032654, -0.04037735238671303, 0.686715304851532, 1.5819028615951538, 0.7723298072814941, -0.64397132396698, 0.3984491229057312, 0.4267870783805847, -0.34168875217437744, -1.1519159078598022, 1.170061707496643, 1.2275676727294922, 0.11143968999385834, -0.14591529965400696, -1.147312879562378, -1.39315664768219, 0.27289143204689026, 1.1960771083831787, 1.4893908500671387, 0.7108557820320129, 0.02186095342040062, -0.5124841928482056, -0.8678469657897949, -0.15574464201927185], [-0.015737341716885567, -0.30336159467697144, 0.5131279826164246, 0.5689161419868469, 0.17869995534420013, 0.40991854667663574, -0.30784252285957336, -0.3142053484916687, -0.7244563102722168, 1.0604963302612305, -1.638069987297058, -1.0747438669204712, 0.4150344431400299, 0.7758029103279114, -0.7164316177368164, 0.9320062398910522, 0.060705989599227905, 0.213626891374588, 1.2289403676986694, -1.5218441486358643, -0.3263312876224518, 0.6816492080688477, -0.910971462726593, 0.7140514254570007, -0.5551586151123047, -0.44005945324897766, 2.1119399070739746, 1.973145842552185, 2.0482289791107178, -0.8646848797798157, -0.20411522686481476, -1.2208248376846313, -0.41819265484809875, 0.9561249017715454, 1.1311932802200317, 0.25104621052742004, -0.4336896240711212, 0.017253853380680084, -0.6517638564109802, 0.12994660437107086, -0.66673344373703, 1.223235011100769, 1.1062991619110107, 1.0725152492523193, -2.651106357574463, -0.3032982051372528, 0.7234923243522644, -1.6354045867919922, 0.5740992426872253, 0.9590516686439514, -1.3239383697509766, -2.1647448539733887, -0.7723048329353333, -0.5227819085121155, 0.49557024240493774, 0.7407382726669312, 0.24162034690380096, -0.5899605751037598, -0.02773481421172619, 0.17675109207630157, 2.187002658843994, 0.6484808921813965, -0.6937018632888794, 0.6990976929664612], [1.126843810081482, -0.24676965177059174, 0.0650029256939888, 0.5427313446998596, 0.29476040601730347, -0.6091870665550232, -0.4898255169391632, 1.8738998174667358, -1.3898621797561646, 1.4803624153137207, -0.8487991094589233, 0.5645252466201782, 0.03609749302268028, 1.4060866832733154, -1.080230712890625, 1.1044126749038696, -0.6222359538078308, 0.5874715447425842, -1.3527330160140991, 1.08790922164917, -0.36914992332458496, -0.2695826292037964, 1.6431334018707275, 0.6126826405525208, -0.3386955261230469, -0.7971678972244263, 0.031224939972162247, -0.6507500410079956, -0.33644720911979675, -0.11342588812112808, 0.1893659383058548, -0.7318815588951111, -0.09789009392261505, 0.6236237287521362, -0.4428679347038269, -0.6211186647415161, 0.8763877153396606, 0.7313156127929688, -0.05000017583370209, 0.4937003254890442, -1.0734180212020874, -0.012179957702755928, -1.5867319107055664, 0.15581899881362915, -0.22150929272174835, 1.2658040523529053, 0.6750698089599609, 0.1027148887515068, 1.6729650497436523, -1.743143081665039, -0.38551613688468933, 0.9552296996116638, -1.2557566165924072, -0.40937644243240356, -1.7598776817321777, 0.6076018214225769, -0.8145566582679749, -1.5093708038330078, -0.03198764845728874, -1.175303339958191, 0.8988187313079834, 0.468168169260025, -0.4494439959526062, -0.03158343583345413], [0.7110010981559753, 2.3512041568756104, 0.3149067461490631, -0.6745606064796448, -0.05329965800046921, 0.7482693195343018, -0.5459433794021606, -1.0687228441238403, -0.3187401294708252, 0.22075670957565308, -0.6901143789291382, -0.8169319033622742, 2.0351388454437256, 1.8123427629470825, -0.5585792660713196, -0.2509465217590332, 0.4168162941932678, -0.8980503082275391, 0.37135806679725647, -1.1144880056381226, -0.44879111647605896, -2.883528709411621, 2.792537212371826, 0.1331232488155365, 0.046434320509433746, -1.221044659614563, 0.08884623646736145, -0.5139676928520203, 1.01068115234375, 0.1211920976638794, 0.63785719871521, -0.6616020202636719, 0.6692846417427063, -1.4027515649795532, -0.4464547336101532, -0.2913404405117035, 0.21950630843639374, -0.3498900234699249, -1.37856125831604, -0.04077048599720001, -1.6491824388504028, 0.4709840416908264, 1.1615067720413208, -0.5221489071846008, -0.9765242338180542, -1.4315909147262573, -0.28754934668540955, -0.6403954029083252, -0.06881330907344818, -0.18083693087100983, -0.23671260476112366, -0.15377168357372284, 0.46116504073143005, 0.9684010744094849, -1.2424190044403076, -0.47589412331581116, 1.0735622644424438, -0.26355263590812683, -0.2637638747692108, 1.246267318725586, -0.03459632024168968, -0.09549888223409653, -0.9002309441566467, 0.5007297396659851], [-0.13352839648723602, -0.5019782781600952, 0.35016846656799316, 1.404880404472351, -0.4567639231681824, 0.3168155550956726, -0.6608524322509766, 0.06583667546510696, -0.3967093825340271, 0.07691667973995209, -0.27224674820899963, 2.0916941165924072, 0.04904831573367119, 1.870676040649414, -1.6445198059082031, 0.24685099720954895, -0.9770547151565552, 0.444275438785553, -1.9810229539871216, -1.5971927642822266, -0.22510680556297302, -0.4177635908126831, -0.1788611263036728, 1.3096787929534912, -2.049546003341675, -1.2551133632659912, -1.9224331378936768, 1.0269365310668945, 0.23335614800453186, -0.8393509984016418, 2.3282508850097656, 0.9807236790657043, 0.8907614350318909, -0.07489783316850662, -0.9250689744949341, 0.4784506559371948, 0.8877821564674377, 0.9609348773956299, 0.17293018102645874, -1.218907117843628, 0.6473337411880493, 0.4728531241416931, 0.3458276391029358, -0.9883437156677246, 0.5787864923477173, 0.6743637919425964, 0.8627797961235046, -0.5773715972900391, -0.4272199869155884, -0.4654795825481415, 1.3244001865386963, 1.0130196809768677, -0.062031738460063934, 0.08485078811645508, -0.17059564590454102, 0.5467352867126465, -0.9485414624214172, -0.4227437674999237, -1.9422920942306519, 0.8552483916282654, 0.7710212469100952, -0.7463023066520691, 1.0531419515609741, 1.0366543531417847], [-1.3828151226043701, 0.5802563428878784, 1.342319369316101, 0.3584554195404053, 0.9000576138496399, -0.5195040702819824, 1.8701056241989136, 1.601969838142395, -1.1523483991622925, -0.8933002948760986, 0.13164789974689484, -0.530426025390625, 1.6479322910308838, -1.4199248552322388, -0.31329962611198425, 0.7792261838912964, -0.4526066482067108, 0.1058046892285347, -0.3349378705024719, -0.3519324064254761, 0.9346946477890015, -0.19527418911457062, -0.5409916639328003, 1.9777652025222778, 1.2546318769454956, 2.1364786624908447, 0.023932084441184998, 1.1831251382827759, -0.7533717751502991, -0.8443590402603149, 0.004523384850472212, 0.5971932411193848, -0.2197258025407791, 0.3479354977607727, 1.1195178031921387, -2.2132532596588135, 1.2599748373031616, 0.023893704637885094, 0.24818289279937744, -0.6943219304084778, 1.4714829921722412, 0.12598609924316406, -0.618539035320282, -0.5772801637649536, -0.6567994356155396, -0.8494839072227478, 0.06920746713876724, -0.8601272106170654, 0.5289168357849121, 0.11819393932819366, 0.8437390923500061, 0.003905319608747959, -1.2759608030319214, 2.220349073410034, -0.28226974606513977, 1.3430438041687012, 0.41644471883773804, 0.007696027867496014, 0.6187416315078735, -0.26056671142578125, -1.9817376136779785, -0.2509341537952423, 1.967386245727539, -1.7200742959976196], [1.2672630548477173, -0.36942893266677856, -1.3863314390182495, 0.30060863494873047, 0.3531067967414856, -2.3275532722473145, -0.9034278988838196, -0.37703338265419006, -0.16618666052818298, -0.9913232326507568, -0.8698661923408508, 0.4818018078804016, 0.3185892105102539, 0.6638206243515015, 0.8831542134284973, -0.36012378334999084, 1.2934824228286743, -1.0842050313949585, 0.9222283959388733, 0.22895751893520355, -0.7413803339004517, 0.31449010968208313, 0.2221778780221939, 2.1012184619903564, -0.27682918310165405, -0.3562796115875244, 0.608936607837677, -0.1948556900024414, 0.6194698810577393, 1.2884210348129272, -2.232149600982666, -0.8363227844238281, -0.13075722754001617, 0.6515058875083923, -0.788017749786377, -1.4499799013137817, -2.7848763465881348, -1.157766580581665, -0.25821271538734436, 1.2264505624771118, 0.8129043579101562, -0.659777045249939, 0.632865309715271, 1.5887669324874878, -0.22570684552192688, 0.34226498007774353, 0.2789766192436218, -0.08363552391529083, -0.030684838071465492, -1.9992091655731201, 1.5750235319137573, -1.2534486055374146, 0.22339539229869843, 0.5826342701911926, 1.4199663400650024, -0.5189005136489868, 0.8913383483886719, 0.326349675655365, -0.03344245254993439, -1.5884613990783691, 0.7160859704017639, -1.6318669319152832, -0.9348002076148987, -0.12315992265939713], [-1.7251490354537964, -0.08107897639274597, -0.5026748776435852, -0.11808709055185318, -0.2782551050186157, -1.925612211227417, 0.3981209993362427, 0.35461536049842834, -0.7328284382820129, 0.7765660881996155, -0.17391958832740784, 1.1841193437576294, -1.277955412864685, -1.8163273334503174, -0.5689924359321594, 2.37198805809021, 0.0960521250963211, 0.6794158816337585, 0.22469089925289154, -0.0584590807557106, -0.15765824913978577, 0.4431832730770111, 0.31945085525512695, -1.1497575044631958, -0.9656638503074646, 1.0735011100769043, 0.30165985226631165, -1.721998929977417, 0.3620430529117584, -0.9631847143173218, 1.7483556270599365, 0.37364518642425537, -0.6867510080337524, -1.6104036569595337, 1.4440302848815918, 1.3295890092849731, 1.500906229019165, -0.10849695652723312, 0.5353899002075195, -1.0054305791854858, 1.7034982442855835, 0.48628121614456177, 0.543773353099823, 0.03523574024438858, -2.5980591773986816, -0.7308408617973328, 1.084360957145691, 1.4979424476623535, -1.1884236335754395, 1.6347194910049438, -0.9732756018638611, 0.8190481066703796, -0.5977665185928345, -0.7278220653533936, -1.476348638534546, 1.8611637353897095, -0.7953505516052246, -0.02549569308757782, 0.5421851277351379, 0.028654756024479866, 0.510793924331665, -0.6523560285568237, 0.1824444979429245, -0.07457537949085236], [0.09609740972518921, -1.0081348419189453, 1.6859294176101685, 0.14159296452999115, 0.16337548196315765, 0.605465292930603, -0.30122366547584534, -0.698660671710968, 0.44362393021583557, 0.07708170264959335, -0.18109454214572906, 1.7433980703353882, -1.5621261596679688, -1.655696153640747, 0.886252760887146, -0.42825788259506226, -0.11893356591463089, 0.08499189466238022, -0.07069215178489685, -0.7614617943763733, 0.03738207370042801, 2.3658463954925537, 0.28963908553123474, 0.9042553901672363, 1.3412790298461914, -1.4594008922576904, 1.5008877515792847, -1.2513453960418701, 0.05801219493150711, 0.8287673592567444, 0.3635902404785156, -0.8280224800109863, 0.7047445178031921, 2.210928201675415, 0.621537983417511, 0.41948843002319336, -0.7922986745834351, 0.030881326645612717, 0.34892401099205017, 1.323033332824707, 0.4177183508872986, -1.0256479978561401, 0.2517041265964508, 1.2648210525512695, -0.5372190475463867, 0.042401302605867386, -0.2916255593299866, 0.26703333854675293, -0.23901066184043884, -0.5740927457809448, -1.5806639194488525, 0.6536623239517212, 0.24266988039016724, -1.0687862634658813, -1.616373062133789, 0.3100840449333191, 0.47938838601112366, -0.6510583162307739, -1.131166934967041, 0.7661281228065491, 1.5733168125152588, -1.8082821369171143, 0.40956106781959534, 0.7110571265220642], [0.8939944505691528, 0.711408257484436, -1.5598310232162476, 1.1568282842636108, -0.9069172739982605, -2.655886173248291, -1.189552903175354, 0.948412299156189, 0.08702097833156586, 0.39227384328842163, 0.3525826632976532, -0.7333253026008606, 0.3110048472881317, -0.1531791388988495, 0.9781601428985596, -0.9033652544021606, 0.910400390625, 0.036194074898958206, -0.2271580845117569, 0.601386308670044, 0.42557454109191895, -1.022400140762329, -0.3081153631210327, -0.025348497554659843, -0.3290543258190155, -0.733348548412323, -0.6285293698310852, -0.09333505481481552, -0.46319612860679626, 2.007920026779175, -0.3171062767505646, 0.5988923907279968, -0.7503212094306946, 0.04999619349837303, -0.5214954614639282, -0.7689312100410461, -0.29968225955963135, -0.847162127494812, -1.169934868812561, -0.49404600262641907, -2.427643299102783, -0.2925514876842499, 1.0574840307235718, -1.0944327116012573, 0.821573793888092, 1.3986103534698486, 0.8329327702522278, 0.10276714712381363, 1.275671124458313, 0.13497211039066315, -1.6701176166534424, -0.20922881364822388, -0.2018006592988968, 0.4698654115200043, -1.4488309621810913, -0.7000150680541992, -1.0499578714370728, 0.6225727796554565, 0.37501925230026245, 0.9165008068084717, 1.2071142196655273, 0.2930399775505066, 1.171433925628662, 1.3730944395065308], [0.5296657085418701, -0.9866903424263, 0.6618480086326599, 0.4039987623691559, -1.165218472480774, -0.8890146017074585, -1.6038334369659424, -0.6811988353729248, 1.9159871339797974, -0.6443542242050171, -0.9738793969154358, -1.8098911046981812, 0.00893403496593237, -0.5831913352012634, 1.751678466796875, -0.6460193395614624, -0.3617931604385376, 1.605252981185913, 1.1744325160980225, -1.0170187950134277, -0.36575332283973694, 0.5510126948356628, -0.7930451035499573, 0.07751326262950897, 0.3736400604248047, 0.5324028134346008, -0.19566203653812408, 0.21500153839588165, 0.29437705874443054, 0.14314605295658112, -0.4468614161014557, 1.9455147981643677, 0.4287523031234741, -0.8572399616241455, 0.4694754481315613, -0.20043720304965973, 0.05153788626194, 0.18711420893669128, 1.9256612062454224, -0.07484693080186844, 0.12092660367488861, 0.13167661428451538, 0.008389653638005257, 0.2922186851501465, 0.3017369508743286, -0.725626528263092, 0.9331875443458557, 2.2677032947540283, 0.1512998342514038, -1.3456814289093018, 0.47346732020378113, -0.19237740337848663, 0.9766160845756531, 0.6837444305419922, -0.6247971653938293, 1.551517128944397, -0.29127463698387146, -0.9215173721313477, 0.10476308315992355, -1.3005386590957642, -2.2759323120117188, 1.1429765224456787, -1.2773857116699219, -1.0162338018417358], [-0.9894539713859558, 0.5611132979393005, 0.16280962526798248, -0.26081058382987976, -0.3043130338191986, -0.599757730960846, -1.1499930620193481, 1.4990299940109253, -0.707977294921875, -0.8732896447181702, 0.12546904385089874, 2.4187910556793213, -0.5960243344306946, 0.32598280906677246, 0.5490704774856567, -1.1036903858184814, 1.1574219465255737, -0.46763724088668823, 0.7905095815658569, -1.4182419776916504, -2.2258083820343018, 0.7916079163551331, -0.971015989780426, -0.8811855912208557, -0.3356533944606781, -1.1975696086883545, 2.0749576091766357, 0.8669931888580322, 0.45256227254867554, -1.1953353881835938, -0.43989697098731995, -0.8519033789634705, -0.2786923348903656, 0.8507711887359619, -0.21400700509548187, -0.8442655205726624, -1.236655354499817, 0.1217375099658966, 0.16277571022510529, -1.4901772737503052, 0.7581027746200562, -0.19527241587638855, 1.0973471403121948, -0.7971606850624084, 1.6570242643356323, -0.44610702991485596, -1.4134159088134766, 0.19870705902576447, -1.492329716682434, 0.4419489800930023, -0.41192376613616943, -0.8960281610488892, -0.9317274689674377, 0.21514977514743805, -0.14302174746990204, 0.026376087218523026, -0.24124915897846222, -1.235119104385376, -1.381584644317627, 0.10475318133831024, 0.28091558814048767, 0.42359524965286255, -0.9750295281410217, 1.2181822061538696], [0.2928358316421509, 0.5808619260787964, -1.519182801246643, -0.53653883934021, 0.8850663304328918, -2.015702486038208, -0.9650530815124512, -1.3534111976623535, -0.6813311576843262, 0.009921266697347164, 1.537976861000061, -0.4662782847881317, -1.278371810913086, 0.2015853375196457, 1.674392819404602, -1.2562299966812134, -0.47634804248809814, -0.4924352467060089, -1.27103590965271, 1.8118973970413208, -1.3902063369750977, -0.18837286531925201, -0.9816763401031494, -0.03519260510802269, 0.006161702796816826, 0.0688813254237175, 0.04190131649374962, -0.1631765216588974, 0.5090900659561157, 0.4508444368839264, 0.7584953904151917, 0.24139940738677979, -1.1661127805709839, -0.23422737419605255, -0.4856615364551544, -0.6486321091651917, -0.8535739779472351, -0.8270182013511658, -2.135530710220337, 2.550839424133301, 0.5137954354286194, 0.06814483553171158, 0.9720386266708374, 0.9412580132484436, -2.7928359508514404, 0.8305160999298096, -1.3872884511947632, 0.4152297377586365, 0.6525433659553528, -0.5521042346954346, 0.34056925773620605, -0.06174541637301445, 0.07028033584356308, -1.6939921379089355, 0.11406921595335007, -0.5027085542678833, -3.2707011699676514, -2.0941250324249268, 1.222949504852295, 1.1025969982147217, -0.2997075915336609, 0.38093703985214233, -0.9999643564224243, -0.5511507391929626], [0.8852744698524475, 1.2698763608932495, 0.6089780926704407, -0.5653830766677856, 1.9382492303848267, -0.7198821306228638, 0.913364589214325, 1.7266372442245483, -0.17587929964065552, -0.4419582486152649, 0.011414187029004097, 1.1986243724822998, -1.4357163906097412, 0.6093676686286926, -1.5756142139434814, -0.0322883240878582, -0.02142954431474209, -1.95219087600708, 0.6092529892921448, 1.7363593578338623, 1.2003308534622192, 2.5146703720092773, -0.8894347548484802, -0.3275081515312195, -1.0051770210266113, 0.10610784590244293, -0.5793353915214539, 1.4166069030761719, -0.6560361981391907, 0.04789767041802406, 0.7616948485374451, -0.9849637746810913, -0.48929640650749207, -1.205152153968811, 0.1420735865831375, -0.10650037229061127, -0.012216072529554367, -0.9845789670944214, 0.8173524141311646, 1.1017141342163086, -0.2106422483921051, -0.8393668532371521, -0.016010968014597893, -0.16719481348991394, 1.2683888673782349, -0.06312083452939987, -0.019229812547564507, -0.02000683918595314, -0.6530421376228333, 0.007552471477538347, 0.39878836274147034, 0.9224133491516113, 1.1129944324493408, -1.4164037704467773, 0.6933364272117615, -0.17622345685958862, 1.6920069456100464, -1.0457898378372192, -2.4708292484283447, 0.3073570430278778, -0.018552647903561592, -0.6106751561164856, -0.531070351600647, -0.7391493916511536], [-0.09007710218429565, 0.15780414640903473, -0.472243070602417, -0.46746137738227844, 0.6947544813156128, -0.5455945730209351, -0.29995137453079224, 1.645801067352295, 0.966529905796051, 0.0710623636841774, 1.1189597845077515, -0.7681159973144531, -0.12638646364212036, 0.13873039186000824, -2.0744426250457764, -0.9687396883964539, -1.4182170629501343, 0.21065925061702728, 0.5773948431015015, 1.4800742864608765, 0.4275098443031311, 0.7520021796226501, 0.695331871509552, 1.6230299472808838, 0.3162863552570343, 0.0014505140716210008, -0.05852580815553665, 2.068661689758301, -1.288988709449768, 1.10693359375, 0.9769005179405212, -0.36869779229164124, -1.0364539623260498, -0.8843043446540833, 1.3968162536621094, 1.6108022928237915, 0.34118619561195374, 0.32934120297431946, 1.106508731842041, 0.6544339060783386, -0.6988745331764221, -0.5393797159194946, -0.14341787993907928, -1.3495087623596191, -0.03443131968379021, 0.02882179245352745, 0.612379252910614, -0.8385955095291138, -0.27905330061912537, -0.11616851389408112, 1.903241753578186, -0.5969874262809753, -1.6454458236694336, -0.9868807196617126, 0.3236153721809387, -1.2565401792526245, 0.4867349863052368, -0.5421774983406067, -1.9275805950164795, 0.45336657762527466, -1.501204490661621, 1.110711932182312, -1.0136570930480957, 0.12525296211242676], [-0.6976339221000671, 0.2527351379394531, 0.4938708543777466, -0.2306281477212906, -0.33546170592308044, 0.8494454026222229, 0.7634371519088745, -1.3301475048065186, -1.3795292377471924, 0.6413091421127319, -0.006078016012907028, -0.5080218315124512, 0.2628307342529297, -1.9041942358016968, 0.05524958297610283, -0.27216246724128723, -0.6827241778373718, -0.30718904733657837, -0.25611618161201477, 1.6492791175842285, 1.1297791004180908, -0.22511063516139984, 1.0260530710220337, 0.08699340373277664, -0.761709451675415, 1.063704252243042, 0.2989329695701599, -0.31407129764556885, 1.9704256057739258, 1.7019685506820679, 1.5896718502044678, 1.1874933242797852, 0.026784226298332214, 1.6924861669540405, -0.6056559085845947, 0.3250786066055298, -0.8051551580429077, 0.47852420806884766, -1.1634788513183594, 2.72729754447937, 0.763667106628418, 0.2223270684480667, 0.40438544750213623, -0.5056801438331604, 0.2399255335330963, 0.23554371297359467, -0.6355903148651123, -0.9743186831474304, 1.6834211349487305, 0.4143744111061096, 1.2517309188842773, 0.07441919296979904, 1.490372657775879, -0.32276901602745056, -0.1351640522480011, -0.065603107213974, 0.7057967185974121, 0.49011075496673584, 0.013171081431210041, 0.6040867567062378, -0.09618601202964783, 1.084963321685791, -0.3416754901409149, -0.40245044231414795], [-1.199679970741272, 1.774356484413147, -0.8053852319717407, -0.597545862197876, -1.1657298803329468, -0.2667683959007263, 0.6293773651123047, -0.18653404712677002, 1.9991291761398315, -0.568252444267273, -1.0150245428085327, -0.5842394828796387, 0.7351705431938171, 1.0528450012207031, 1.7272911071777344, -0.05348280072212219, -0.2953333556652069, -0.02632269263267517, 0.38166293501853943, -1.2821677923202515, -1.030288577079773, 0.42460867762565613, -0.5798379778862, -1.8210376501083374, 0.8786439895629883, -0.7870298624038696, 0.14836007356643677, 0.45788368582725525, -1.15988028049469, 1.2983356714248657, -1.0119552612304688, 0.8692296147346497, -1.4405338764190674, -1.0147945880889893, 1.3294599056243896, -0.16309866309165955, 0.45689499378204346, -0.28992852568626404, -1.9037467241287231, 0.5548738837242126, 0.06985427439212799, -1.8491299152374268, -1.9340041875839233, 0.5347850918769836, -0.22692640125751495, -0.504673182964325, 0.8374954462051392, -0.185263529419899, 0.588335394859314, -1.340914249420166, -0.9524668455123901, -0.9874972105026245, 0.22437313199043274, -0.4927632808685303, 1.0203967094421387, 0.3206242024898529, -0.5687857866287231, 0.5145434737205505, 1.1259199380874634, -2.0843863487243652, -1.957011103630066, 0.21760372817516327, -0.2585752606391907, 0.4721464514732361], [-1.219297170639038, 0.4991428256034851, 0.40186670422554016, -1.1176207065582275, 2.0194618701934814, 0.05228515341877937, 0.6818576455116272, -1.21858811378479, -0.6475342512130737, -0.030356084927916527, 1.1698262691497803, -1.8278255462646484, -0.592322587966919, -0.23332734405994415, -0.7300803065299988, 0.417758971452713, -0.28926506638526917, 0.2850646674633026, -1.0219095945358276, -1.521803855895996, 0.562147855758667, 0.22345800697803497, -1.4938970804214478, -0.7560986876487732, -0.25653550028800964, -0.22040902078151703, -0.12425623089075089, -1.1891484260559082, 0.38957199454307556, 0.487865149974823, 2.02341365814209, -0.3570849597454071, -1.9358946084976196, 0.7741852402687073, 0.8483986258506775, 0.8276845812797546, -0.5025049448013306, -0.14258190989494324, -0.17800013720989227, -0.13984248042106628, -0.658958911895752, 0.6495478749275208, 0.25854846835136414, 1.143884301185608, 0.10849414765834808, -0.3195191025733948, 0.9196382761001587, -1.458985447883606, 0.2820736765861511, -0.07000325620174408, -1.5078626871109009, -0.8481171727180481, -0.7145698070526123, -0.7453242540359497, -0.47829726338386536, 1.5451356172561646, 0.5543113350868225, 0.10056596249341965, -1.5531753301620483, -1.613326072692871, 1.37253737449646, -0.03963352367281914, -1.1516112089157104, 0.7624281048774719], [0.6131635308265686, -0.844684898853302, -0.7528887391090393, -0.38221704959869385, -0.46405860781669617, 1.6413922309875488, -1.5245906114578247, 0.2022366225719452, -0.8237506151199341, -1.5209877490997314, -1.4473600387573242, 0.6592951416969299, 0.3090197741985321, 0.33505553007125854, -1.703550100326538, -0.14456021785736084, -1.0666338205337524, 1.1860915422439575, -0.8678868412971497, -0.40877217054367065, 0.8286117315292358, 0.5615593791007996, 0.1128239706158638, 0.9230523109436035, 0.23450659215450287, -0.5945033431053162, 0.15307864546775818, -0.17138849198818207, -0.8179791569709778, -0.8232437372207642, 1.4312026500701904, -1.2440515756607056, -2.418501138687134, -0.6990485191345215, -0.7287991642951965, -1.6060482263565063, -0.853395938873291, -0.6662248373031616, 0.4312129318714142, 1.1805306673049927, -0.851018488407135, 0.8688064217567444, 1.194875955581665, 1.4709970951080322, 0.4642563462257385, 0.9386016726493835, 0.12636157870292664, -0.1656825691461563, -1.893770694732666, 0.2391837239265442, 0.5578635334968567, -0.07662045955657959, 2.3526766300201416, 0.26694878935813904, 0.7612957954406738, 0.2385992556810379, 0.14932431280612946, 1.009825587272644, 0.631645917892456, -0.1973726451396942, 0.6783617734909058, 0.07575488090515137, -0.7293532490730286, -0.7266444563865662], [-0.9592899680137634, -0.7238327860832214, 1.7066465616226196, -1.0510072708129883, -2.060107707977295, 1.8048763275146484, 0.8871620297431946, -0.7382245659828186, -1.5700575113296509, -0.676385760307312, -1.00993013381958, -1.312497615814209, -0.6945138573646545, -0.6468108892440796, -0.0403667613863945, 1.5055913925170898, 0.5087373852729797, -1.9008617401123047, 0.9311656355857849, 0.7953787446022034, -0.6998544335365295, -0.7396212816238403, 1.4186336994171143, -0.5313606262207031, 1.4396069049835205, 1.0425338745117188, 0.19584473967552185, -0.42200660705566406, -0.4705694913864136, -1.3050819635391235, -0.7030340433120728, 1.6130925416946411, -0.8829421401023865, 0.40586698055267334, -0.20022878050804138, -0.2389877587556839, -0.5316148996353149, -0.056803952902555466, -0.681900680065155, -1.2477648258209229, 0.8184161186218262, 0.03729958087205887, 0.30304425954818726, 0.3803271949291229, -1.240121841430664, -0.9400814771652222, -0.15027646720409393, 0.18150736391544342, 0.11465460062026978, 0.44064250588417053, 1.7961176633834839, -0.004630897659808397, -1.6104416847229004, 0.9718458652496338, 0.17302998900413513, -0.4228435456752777, -0.6797916889190674, 1.5566297769546509, 0.21030914783477783, 0.3568786084651947, -0.9718781113624573, 1.189882755279541, -0.9787778854370117, -0.4559497833251953], [0.7003921866416931, 0.7257806658744812, -0.9100826382637024, -0.6819656491279602, -0.3534373939037323, 2.7165825366973877, 0.4349900782108307, 0.5399531722068787, 0.03905807062983513, -0.2841989994049072, -0.028626779094338417, -0.8789215683937073, 0.12276806682348251, -0.8835535645484924, 1.5299736261367798, -1.029930591583252, 1.562171459197998, 0.09994903206825256, -1.3309434652328491, 1.0827819108963013, -1.4012936353683472, -0.6414247751235962, 0.28398263454437256, 1.0947227478027344, 1.877898097038269, -1.592647910118103, 0.8174609541893005, 1.1572891473770142, -0.18714988231658936, 0.7099916338920593, -0.2753455936908722, -0.5470176339149475, -0.5691186189651489, -1.0631608963012695, 0.5406228303909302, 0.8801099061965942, 1.442138671875, -0.49423888325691223, -0.1406109780073166, -1.396836757659912, -0.5790295004844666, 0.36157935857772827, -1.082910180091858, -0.15918616950511932, -0.054834846407175064, -1.3959641456604004, -0.2970220446586609, -0.0907793715596199, -0.5292069911956787, 1.0873247385025024, -0.38672998547554016, -0.42441511154174805, -0.5946576595306396, -1.9057774543762207, 0.9352563619613647, -0.2205866128206253, 0.23220065236091614, 1.0268723964691162, -0.08292175829410553, 0.13073062896728516, 0.25838080048561096, 0.921734094619751, 0.42961356043815613, -0.795391857624054], [-1.3033106327056885, 1.4066604375839233, -1.2820457220077515, -0.03985704854130745, 1.5152711868286133, 0.48971936106681824, -0.8267080187797546, -0.4218875765800476, 0.06991781294345856, -0.6665635108947754, -0.38952961564064026, -1.7989414930343628, 1.2149373292922974, 0.5028377771377563, 0.8970596790313721, 0.33294641971588135, 1.3232686519622803, -0.8950625061988831, -0.9083831906318665, 1.1638245582580566, 0.29427194595336914, 0.422040194272995, 0.11153741925954819, -1.249462604522705, 0.19730347394943237, -0.054725080728530884, -0.20817552506923676, -0.6890735030174255, -1.2128326892852783, -0.33101576566696167, 2.0180819034576416, -0.2131459265947342, -0.5613369345664978, -0.8123265504837036, -1.2733062505722046, -0.5256058573722839, 0.3041631281375885, -0.2150854766368866, 0.9090269804000854, -1.0142384767532349, 0.38938310742378235, -0.9200571775436401, -1.0953900814056396, 0.7027201652526855, 0.09373914450407028, 0.36642953753471375, -0.7894278168678284, -0.2350536584854126, 0.473509818315506, -0.18893083930015564, 1.4801528453826904, -0.3813728988170624, 0.4098801612854004, -0.34251078963279724, -0.425598680973053, 0.04513707011938095, -0.16756099462509155, 0.33456727862358093, -0.25849366188049316, 0.4781130850315094, -1.9057692289352417, -0.43551090359687805, 0.5662127137184143, -0.08602029085159302], [1.1259868144989014, 0.6046069860458374, 0.9230302572250366, -0.8080509901046753, -0.9519257545471191, 2.002737522125244, 1.4784595966339111, -0.5299063324928284, 1.7481135129928589, -0.5869644284248352, -0.5111430883407593, -0.31888547539711, 0.6351559162139893, -1.4203301668167114, -0.6712257862091064, 0.22924764454364777, -1.626299500465393, 0.8028342127799988, -0.9863670468330383, -1.303837537765503, -0.7217611074447632, 0.5169395208358765, -0.1842944473028183, -2.038182020187378, 1.1118409633636475, 0.643517792224884, 0.40621140599250793, 0.7663386464118958, -0.4238710105419159, 1.4191279411315918, -0.8960840106010437, 1.3725099563598633, -0.13115809857845306, 0.287832111120224, 0.7299439907073975, -0.07710912078619003, -0.5370124578475952, -1.9924579858779907, 0.4274665415287018, 0.5877487063407898, 0.6195710897445679, 1.0024741888046265, 1.0224088430404663, 0.834098756313324, -0.11579549312591553, -0.45608001947402954, -1.2875033617019653, -0.2533550560474396, 1.3377233743667603, 0.5324805974960327, -1.312905192375183, -0.4063974916934967, -0.269976407289505, -0.6835232973098755, 2.3484864234924316, -0.33600834012031555, 0.40230533480644226, 0.7910738587379456, 0.3299409747123718, 1.7108874320983887, 0.38050854206085205, 0.10096053779125214, -0.16914786398410797, 0.37747788429260254], [0.841856837272644, -1.1450339555740356, 0.9444769024848938, -2.1703431606292725, -0.8922801613807678, 0.8002980351448059, -0.31437358260154724, 0.4792851507663727, -0.2225622534751892, -1.228681206703186, -0.6255533695220947, -0.10445354878902435, 1.5208261013031006, 0.565686821937561, 1.0463308095932007, 2.426495313644409, -1.3006887435913086, -1.8017481565475464, -0.4455336034297943, 0.23562441766262054, 0.9139465093612671, 1.1543446779251099, -0.9218987226486206, 0.5320351719856262, -0.8650230169296265, -1.7722737789154053, 0.4508688151836395, 1.4012142419815063, 0.03048802725970745, 0.12353114783763885, 2.263558864593506, 1.3964329957962036, 1.2932714223861694, 1.409531831741333, 0.5022245049476624, -0.6418019533157349, 0.3877141773700714, -0.3002289831638336, 0.6794949173927307, 1.2014256715774536, 0.9941251277923584, -2.101097822189331, -0.6672578454017639, -1.3018155097961426, -0.7647889852523804, -1.6785951852798462, 1.220441460609436, -1.6097652912139893, 0.5935717225074768, -1.023708701133728, -0.9237040281295776, 1.635414481163025, 1.4364699125289917, 0.0573577955365181, 0.03606748580932617, -1.3616856336593628, 1.5136160850524902, -0.5883947014808655, 0.3261972665786743, -1.5362292528152466, 1.6435106992721558, 1.044072151184082, -0.6279465556144714, -0.9761809706687927], [-0.28570565581321716, 0.8838898539543152, 0.5786072015762329, 0.9754009246826172, 0.4374659061431885, 0.008251853287220001, 0.08485639095306396, -0.4190020263195038, -0.513523280620575, -0.25272679328918457, -0.8214368224143982, 0.40825775265693665, 0.8617091774940491, 0.3083023726940155, -0.9103569984436035, -0.8737043142318726, 1.1372172832489014, -0.1885654777288437, 1.5054751634597778, 0.42516785860061646, -1.883644938468933, -0.9592452049255371, -0.6842378377914429, 0.439891517162323, -0.7595685124397278, 0.8653849363327026, 1.415518045425415, -0.1631651669740677, 0.05700094252824783, -2.3131539821624756, 1.558099627494812, -0.6825527548789978, 0.3668360114097595, 0.7975472211837769, 0.988720715045929, -0.36252886056900024, 0.4143299460411072, -0.0334760881960392, -0.22866538166999817, 1.335962176322937, 1.6938263177871704, -0.9266980886459351, -0.5182758569717407, -1.7199102640151978, 0.5522729158401489, 1.0941191911697388, -0.01582791842520237, -0.5925688147544861, -0.5082014203071594, 2.156426191329956, 0.7159436345100403, -1.5991469621658325, -0.23713736236095428, -1.230419635772705, -1.2267109155654907, -0.5625470280647278, -0.18601618707180023, 0.6918821334838867, -1.6340073347091675, -0.6096293330192566, 1.7857515811920166, -1.0707470178604126, 0.4927982985973358, 1.3060719966888428], [0.40928980708122253, -2.5677084922790527, 0.002882137894630432, -0.7429865598678589, 0.44946610927581787, -0.6206575036048889, -0.6828873157501221, -0.23132239282131195, -1.6398311853408813, -1.619531512260437, -0.056224990636110306, -0.6774577498435974, 0.15984191000461578, 1.6022297143936157, -0.8008543252944946, 0.6281436681747437, 0.7562488317489624, 0.3932713270187378, -0.15303853154182434, 0.042120616883039474, 1.291718602180481, -0.9270778894424438, 0.5428522229194641, -1.1135836839675903, -0.7270276546478271, -1.5502530336380005, 0.8809636235237122, -0.747449517250061, -1.0178345441818237, 0.8826913833618164, -0.32475027441978455, 0.6870594024658203, 0.5065077543258667, 0.6705136895179749, -1.6219960451126099, -0.5346101522445679, -0.011146178469061852, 0.20787271857261658, -0.13798260688781738, -1.0211087465286255, 0.6171303987503052, -1.632682204246521, 1.2926363945007324, -0.0873798355460167, 2.1751034259796143, -0.4910428524017334, -0.11425907164812088, -1.4277758598327637, 0.7730377316474915, 0.3349669277667999, -0.5492316484451294, 0.026134731248021126, -0.05424720421433449, -0.4097428619861603, -1.093353509902954, -1.7074596881866455, -0.5553503632545471, -0.7643505930900574, 0.012371903285384178, -1.1882636547088623, 0.05553160980343819, 1.0363292694091797, 0.8832557797431946, -0.7477616667747498], [-0.5520423650741577, 1.0859886407852173, 1.3225258588790894, 0.268728643655777, -1.4994779825210571, 1.1720161437988281, 0.571181058883667, 0.8883200287818909, -0.4349859952926636, -1.5879119634628296, -0.013963811099529266, -1.6462804079055786, 0.3726607859134674, -0.4720431864261627, -0.7493882179260254, 0.16285710036754608, 2.8043181896209717, 0.3066250681877136, 0.3194241225719452, -0.5462250709533691, 0.9538364410400391, 1.4376906156539917, -0.7429673671722412, 0.15333348512649536, -0.2964063584804535, 1.9743365049362183, 0.4526362717151642, 0.6274906992912292, -0.19538624584674835, 1.1016913652420044, 0.20459245145320892, 0.29672202467918396, -0.1432986855506897, 3.3788323402404785, -0.138039693236351, -0.8541596531867981, 1.4263463020324707, -1.04533052444458, -0.14082668721675873, -0.7056553959846497, -0.3336378037929535, -0.2520011067390442, 1.853455662727356, 0.3915071487426758, 0.6857022047042847, -0.841759204864502, -1.368916630744934, -0.004415035713464022, 0.8459863662719727, -1.2039018869400024, -0.24549800157546997, -0.7037305235862732, 0.11718697845935822, -0.43550437688827515, -0.3640408217906952, -0.12851670384407043, -0.3649766445159912, 0.08235947787761688, -1.1896111965179443, 1.4610121250152588, -0.9188774228096008, 1.1869239807128906, 1.5223320722579956, 0.3733212947845459], [-0.6893441081047058, 0.03536004200577736, -0.4310115873813629, -1.27858304977417, 0.3659292757511139, 0.3648194372653961, 0.261134535074234, 0.6060419678688049, 0.7187613844871521, -0.3030060827732086, -1.0506511926651, -1.0167425870895386, -0.004539186134934425, 0.5297364592552185, -0.9453564286231995, -0.12921081483364105, -1.496098518371582, 1.5520415306091309, 0.4977743327617645, -0.39888060092926025, -0.9335992932319641, -0.6963770389556885, 0.9960245490074158, -0.05879297852516174, -0.2695154845714569, 0.08943842351436615, -0.46519935131073, 1.3463478088378906, 0.33650076389312744, 0.2086559385061264, -1.7411930561065674, 0.2457948625087738, 0.7190005779266357, -0.8356059193611145, -1.3490277528762817, 0.8583317399024963, -1.0688084363937378, 0.4380126893520355, -0.07025446742773056, 0.2858797311782837, 0.378341406583786, 0.23989132046699524, 2.761366367340088, -0.613158643245697, -0.12922817468643188, -1.4968518018722534, -0.14237742125988007, -0.7146204710006714, -1.1133463382720947, 0.342012882232666, -0.36613425612449646, -0.8639280200004578, -0.25765717029571533, -0.7386599183082581, 3.7896647453308105, 0.006414669565856457, -0.15674200654029846, -0.5338376760482788, -0.3899036645889282, -0.20414841175079346, 1.2914323806762695, 0.5403353571891785, 1.5300991535186768, -0.6250133514404297], [-0.6723766326904297, 0.035759229212999344, -0.2255128026008606, 1.41484797000885, -0.7403748035430908, -1.3195165395736694, 1.3821238279342651, -1.5408324003219604, -0.7558021545410156, -1.1321766376495361, 2.2704038619995117, -0.9091241359710693, -0.7860302329063416, -1.1796241998672485, -1.4165340662002563, 0.16531695425510406, -1.662218451499939, 1.2151522636413574, 1.4271806478500366, 0.3633492588996887, -0.38642069697380066, 0.4413364827632904, 1.5382883548736572, 0.6151176691055298, -1.9672060012817383, -0.4985232651233673, -0.20783331990242004, -1.2945302724838257, -1.526906132698059, 0.9097630381584167, -0.28561997413635254, -0.9031352996826172, 2.028378486633301, 0.37351125478744507, 1.5673694610595703, -0.05538899451494217, 0.5643608570098877, 1.6041301488876343, 0.2983156144618988, 1.3623299598693848, -0.3473755419254303, 0.09765167534351349, 0.9204035997390747, -1.2124574184417725, 1.2745634317398071, 1.8774970769882202, -1.5626789331436157, 1.4551432132720947, 0.07881861925125122, -0.9654240608215332, 0.1722121387720108, 0.3428777754306793, -1.2472211122512817, 1.5341424942016602, -0.5174989700317383, 0.2851378917694092, -0.4923519194126129, 0.9838768243789673, -0.9774981141090393, -0.17224812507629395, 0.29411494731903076, 1.126685619354248, -1.8794809579849243, -0.5453296303749084], [-0.2130049616098404, -1.2435328960418701, 0.4123644232749939, -1.4059429168701172, 0.9917994141578674, 0.2694730758666992, -1.401496410369873, 1.539335012435913, -0.13073650002479553, -2.202064275741577, -0.07817631959915161, -0.6043229699134827, 1.236193299293518, 0.3611186742782593, 0.01709211803972721, 1.2510923147201538, -1.3720381259918213, -1.5828828811645508, 1.1876195669174194, -0.8431320190429688, 1.0477076768875122, -0.9470675587654114, -0.5955442190170288, -1.827662467956543, -0.2812991738319397, -2.0204339027404785, -2.050692558288574, -1.1438966989517212, 1.3686796426773071, 2.237999439239502, -1.1047320365905762, 0.08918315172195435, 0.7217774987220764, -1.1502513885498047, 1.169848918914795, 0.056230586022138596, 1.5710632801055908, -0.7956089973449707, 1.2163748741149902, -0.3525477945804596, -0.5509551167488098, -0.8779939413070679, 0.9424623250961304, -0.8603179454803467, -0.9066709876060486, -0.6939782500267029, -0.7086432576179504, -0.28906041383743286, 1.5080945491790771, -0.5032057166099548, 0.6379900574684143, -1.1994454860687256, -1.0359110832214355, -0.7784467935562134, 0.7734938859939575, -0.7525455355644226, 0.0485377162694931, 0.8342557549476624, -1.212874412536621, -0.41317465901374817, 0.05269157886505127, -0.8433148860931396, -0.5257390141487122, 0.8889737725257874], [-0.6827538013458252, 1.0515732765197754, 0.19684767723083496, 2.3473362922668457, 0.1639600694179535, 0.7914035320281982, 1.6893197298049927, 0.8155768513679504, -0.49501216411590576, 0.7995867133140564, 1.828896403312683, -0.5805046558380127, -1.333829402923584, 0.7254712581634521, 0.26466625928878784, -1.2568583488464355, 1.1369701623916626, -0.7501211166381836, 1.1866244077682495, -0.06608438491821289, 0.2331201136112213, 0.05890514329075813, 1.0751553773880005, 0.5545533299446106, 0.09094267338514328, -0.47417619824409485, -0.38965827226638794, -0.37255629897117615, -1.1290185451507568, 0.7949648499488831, 0.6427140235900879, -0.4477063715457916, -0.678856611251831, -0.536812424659729, 0.23373113572597504, -0.8730323314666748, -0.6498338580131531, 1.609916090965271, -0.20328272879123688, -0.45900389552116394, -0.00044868633267469704, -1.3786643743515015, -2.60843563079834, -0.39409980177879333, 1.2532258033752441, 0.29640015959739685, -1.1588398218154907, 1.0525870323181152, 0.6703192591667175, -0.6043822765350342, 0.9906357526779175, -0.9051440954208374, 0.03756579011678696, 0.7406315803527832, -0.8868589401245117, -0.5626034140586853, -0.4676312804222107, -1.1429007053375244, 1.0062634944915771, 1.153695821762085, 1.3335078954696655, 0.03090113215148449, 0.3914664685726166, 1.0794918537139893], [0.40231478214263916, -1.3645235300064087, 1.587083101272583, 0.01494444627314806, 0.08928484469652176, -1.1201940774917603, -1.408516526222229, 0.8699895143508911, 0.3929812014102936, -0.619744062423706, 1.2234773635864258, 0.04788374528288841, 1.3630647659301758, -0.8720300793647766, -0.2080804854631424, 0.6463603377342224, 0.3709700107574463, 1.147415280342102, 0.7184569239616394, 0.864311158657074, 3.1206674575805664, 0.001137211686000228, 0.2719365954399109, 0.8551873564720154, 0.0365665927529335, 0.4910004436969757, -1.0009528398513794, -0.5210745930671692, -2.0966012477874756, -0.06492003053426743, -0.42001405358314514, -1.4222322702407837, 0.29440420866012573, -0.6356819272041321, -0.9350640177726746, 0.1931387484073639, 0.2532272934913635, 1.3458874225616455, -0.8016520142555237, 1.6435248851776123, 1.2268257141113281, 2.1439619064331055, 0.9356425404548645, -0.485102117061615, 0.2550988495349884, -1.294135570526123, -0.7936386466026306, 1.0805678367614746, -0.22049470245838165, -1.168766736984253, -0.07871678471565247, 1.690639853477478, -0.8279121518135071, 0.5509355664253235, 0.11507365852594376, -1.3687844276428223, -0.41619032621383667, 0.0036904450971633196, -0.08381595462560654, 0.5895370244979858, 1.0190980434417725, 0.5489601492881775, -0.9526585936546326, -1.0597642660140991], [0.7608826160430908, 2.8184762001037598, 0.06272342801094055, 0.7187570333480835, -0.38449758291244507, 0.8528684973716736, 0.36239296197891235, -0.354300320148468, 0.30434539914131165, -1.6460237503051758, -0.02004741132259369, -1.2907679080963135, -0.7333559393882751, 0.1885925531387329, 0.9688736200332642, 0.03349587321281433, -1.2192928791046143, 0.08447976410388947, 0.8690597414970398, 0.6056842803955078, 0.3263620436191559, -1.026990294456482, -1.5088927745819092, -0.06874259561300278, 0.4398438036441803, -0.8320719003677368, -0.8268362283706665, -1.1231462955474854, 0.826743483543396, 0.4601963460445404, -0.4082771837711334, -1.4643806219100952, 0.7510669827461243, 1.0652364492416382, 0.38043123483657837, 0.6688551902770996, 1.7629033327102661, -1.3440508842468262, 0.32115423679351807, -1.3455607891082764, 1.4480844736099243, 0.6485865712165833, 0.42163437604904175, 0.40161561965942383, 0.10158741474151611, 0.5868804454803467, -0.6394264698028564, 1.8258254528045654, 1.6191859245300293, 0.6932895183563232, -0.19924764335155487, 0.10433237254619598, 0.27927106618881226, 1.3787193298339844, 0.48767417669296265, 0.4260982275009155, -0.03092687390744686, -0.41251125931739807, -1.1010164022445679, -0.008747025392949581, -0.7548929452896118, 0.7741004824638367, -0.6850314140319824, 1.0071947574615479], [-0.6903495192527771, -1.6768221855163574, -0.5141274333000183, 0.8297008275985718, 1.3120172023773193, 1.6964912414550781, 1.812262773513794, 0.2560517191886902, 1.4450252056121826, -0.205598846077919, -0.40735548734664917, 0.30838868021965027, -0.30331069231033325, 1.1852946281433105, -1.0660104751586914, -0.532249927520752, -0.6761131286621094, 1.1091545820236206, -1.0009214878082275, 0.5462714433670044, -1.1789976358413696, -0.3073200583457947, -1.6090251207351685, 1.9259618520736694, 1.9753522872924805, -0.18666431307792664, 1.0296396017074585, 0.2728244364261627, -0.9771549701690674, -1.4597351551055908, -1.031473159790039, 0.6473228931427002, 3.5949549674987793, -0.466398149728775, -0.7578229308128357, -0.5633202195167542, 0.03256768733263016, 1.339368224143982, 1.0760602951049805, -0.4401107430458069, 1.4874728918075562, -0.2688325345516205, -2.511807680130005, -0.07871613651514053, -0.8438553214073181, 1.6232600212097168, 1.1029388904571533, -0.8654105067253113, 0.20946086943149567, -0.08019759505987167, 0.3628201484680176, -0.39468738436698914, 0.6973878741264343, 1.684952974319458, -0.5126886963844299, -0.4374299943447113, -1.4348454475402832, 1.4333430528640747, 0.7314529418945312, -0.13726447522640228, -1.143704891204834, 0.7319345474243164, -1.4297220706939697, -0.12453897297382355], [-2.1281654834747314, 0.9138179421424866, -0.42807477712631226, -1.5749561786651611, 0.8742086887359619, 0.0874996930360794, -1.7612565755844116, -1.0199707746505737, 1.1542335748672485, 1.673179030418396, -0.27960914373397827, 0.8370154500007629, -0.9433608651161194, 2.003089427947998, -0.4430938959121704, 0.7982232570648193, 0.8642737865447998, -0.01934603601694107, 1.7352628707885742, 0.024883229285478592, -0.34818705916404724, -2.0345311164855957, 1.2531074285507202, 0.4278109669685364, -1.290520429611206, 2.193195104598999, -0.4075651168823242, -0.7331520318984985, 0.7166181802749634, -0.23495854437351227, -0.23570260405540466, 1.714219570159912, -0.09494689106941223, -0.39579880237579346, 0.06653641164302826, 0.5504438281059265, -0.7662435173988342, 0.3164953589439392, 0.5148665308952332, 0.769012451171875, -0.8704279661178589, 1.1727657318115234, -1.2098362445831299, 0.8531339168548584, 1.1571744680404663, -1.2814598083496094, 0.3161107003688812, -0.195419579744339, -0.4592866599559784, 0.3141157925128937, 0.2712344229221344, -0.7018914222717285, 1.178884506225586, 0.6836831569671631, -0.4374762177467346, -0.19834518432617188, -0.47543278336524963, 1.7646257877349854, -0.5963066220283508, 0.6879705190658569, 0.11676561832427979, 0.030241571366786957, 1.3170863389968872, -2.1795108318328857], [1.260977864265442, 0.0690164640545845, -0.7068982720375061, -0.41520532965660095, 0.3384298086166382, -0.6637035012245178, 1.1923046112060547, 0.238239586353302, 0.6391605138778687, 1.3518730401992798, -0.8414399027824402, 0.47948095202445984, -0.24543115496635437, -0.4848749339580536, 0.5484453439712524, -0.07770861685276031, -0.6104666590690613, -1.7359604835510254, -1.144650936126709, 0.4105794429779053, 1.1426209211349487, -0.0021119124721735716, -0.1525821089744568, -0.7947142720222473, -0.2976139485836029, -0.4464119076728821, 0.3454841673374176, 0.32228320837020874, -1.7602224349975586, -0.23474332690238953, -0.6804088354110718, 0.5439611077308655, 0.871473491191864, -0.8197897672653198, -0.05054780840873718, 1.2163078784942627, -1.7598425149917603, -0.8283255696296692, -0.899149477481842, -0.8752253651618958, 0.27631714940071106, 0.9146451950073242, 0.9919556975364685, 0.531582772731781, -0.3423389494419098, -0.3150346577167511, -1.3749362230300903, -0.8547372817993164, 0.7507753968238831, -0.684775173664093, 0.6848353147506714, 0.7325833439826965, -0.5690020322799683, 0.27867352962493896, -0.6321259140968323, -0.3291690945625305, -1.855417013168335, -0.11390133202075958, -0.3952251374721527, -0.41393333673477173, 0.22544682025909424, 1.0383745431900024, 2.060299873352051, -0.10013169795274734], [-0.6404725909233093, -0.8274314999580383, -0.8418830633163452, -0.7898131012916565, 1.3325711488723755, -0.3572155237197876, 0.11258821189403534, 1.3966301679611206, 0.24579842388629913, 0.5292003750801086, 1.1162165403366089, 0.09924047440290451, -0.9532126188278198, -0.487571120262146, -0.4477355182170868, 0.20424684882164001, 0.4193870425224304, -1.1569267511367798, -0.7137082815170288, 0.7411359548568726, 1.1977585554122925, 0.6140614748001099, -0.23386454582214355, -0.4226476550102234, -1.174821376800537, 1.3517045974731445, 0.6529694199562073, 0.1229826956987381, 0.5325093865394592, 0.8215263485908508, 0.48294615745544434, -0.34338515996932983, 1.8947854042053223, -0.41033777594566345, -1.1162890195846558, 0.48198363184928894, -1.6330549716949463, 1.4077309370040894, 0.5143483281135559, 0.21833071112632751, -0.7738755345344543, -0.30960318446159363, -1.333350419998169, 0.04133250191807747, 0.49442410469055176, 1.743552327156067, -0.28219109773635864, 0.9873917102813721, 0.4757237434387207, 1.3226734399795532, 1.3720487356185913, 0.7868326306343079, 0.6143631339073181, -1.2390072345733643, -0.24890238046646118, 0.3093222975730896, 0.39621302485466003, 0.848281979560852, 1.2513540983200073, -1.0417637825012207, -0.29872217774391174, -0.9954853653907776, -0.7849984169006348, -0.07371051609516144], [-0.22186318039894104, 1.5120279788970947, -0.663424551486969, -0.894040584564209, -3.1509997844696045, 0.21147477626800537, 0.16341358423233032, -1.0311461687088013, 0.8539716601371765, -1.6475154161453247, 0.9323357343673706, -0.604084312915802, -0.8442041873931885, 0.46292540431022644, 0.38886329531669617, -0.019496800377964973, 1.3152414560317993, -0.48942556977272034, 0.6047492623329163, 0.291515588760376, 0.08680270612239838, -0.2822912931442261, -0.753036379814148, -0.14577405154705048, -0.9241253733634949, 0.6787844896316528, -2.681910753250122, 0.35106292366981506, -1.1707804203033447, 0.7627179026603699, -1.653099536895752, 0.9558864235877991, -0.41736581921577454, 0.6486005783081055, -0.8679965734481812, -0.6971336007118225, 0.03331025689840317, 1.1026732921600342, -1.0490329265594482, 1.335741639137268, 2.8094236850738525, 0.5818295478820801, -0.12392158061265945, 0.9432442784309387, 0.012104975990951061, 0.4270925521850586, -0.8068810105323792, -0.10547082871198654, -0.3488943874835968, 0.12298262864351273, 2.373311758041382, -0.40207958221435547, -0.8782449960708618, 1.4984172582626343, -1.6937226057052612, -1.1559382677078247, -0.4166017770767212, -0.37129831314086914, 0.4281142055988312, 1.141932725906372, -0.16144369542598724, -0.02317827381193638, 0.6367643475532532, -0.32682594656944275], [-1.6992981433868408, -0.9594389796257019, 0.00810338743031025, -0.038201674818992615, 0.8393438458442688, 0.8742787837982178, -0.004039326682686806, 1.1155365705490112, -0.34722256660461426, -1.4411580562591553, -1.350392460823059, -0.9196447730064392, 0.966788649559021, -1.4679443836212158, -0.6745753288269043, -2.2166502475738525, -0.32760652899742126, -0.45139196515083313, 0.9399033784866333, -0.6668039560317993, -0.8011031746864319, -0.4015757143497467, -1.1748536825180054, -0.3516162931919098, -0.06768302619457245, -1.3660380840301514, 0.9385077953338623, 1.3568257093429565, -0.9064638018608093, 1.0133970975875854, 0.5680108070373535, 0.7832220196723938, -0.17568090558052063, -0.6050534844398499, 2.1212825775146484, 0.14423337578773499, 0.8208902478218079, -1.409323811531067, -1.3469585180282593, -0.04114518687129021, 2.6903209686279297, -1.7178627252578735, 0.6197727918624878, 1.6108530759811401, -1.1826934814453125, 0.7632675170898438, 0.9271316528320312, 0.5568808913230896, 0.13333861529827118, -0.23630447685718536, -0.02302195318043232, 0.8127471208572388, 0.8959407210350037, -0.9505214095115662, 0.32874932885169983, -0.14511916041374207, -1.0860127210617065, -0.13685764372348785, -0.6248524188995361, -0.0903487429022789, 1.8571027517318726, 0.5704109072685242, 1.291939377784729, -0.9816812872886658], [1.922843098640442, -0.770548403263092, -0.5461075305938721, 0.032646920531988144, 0.6789682507514954, 0.4147307872772217, 0.6110007166862488, -1.3683555126190186, -0.020102694630622864, -0.3682595193386078, 1.3148763179779053, -0.2973463833332062, 0.7830062508583069, -0.6259347796440125, -1.6273455619812012, -0.5000970363616943, -1.6378637552261353, -0.24627423286437988, 1.0398226976394653, 1.554581642150879, -0.6296624541282654, -0.16529667377471924, 0.28408411145210266, 1.3234877586364746, 1.6814581155776978, -0.07477834075689316, 0.06770967692136765, 1.0135048627853394, -1.5461112260818481, -0.1578822135925293, -0.11603527516126633, 1.0468753576278687, 0.33318936824798584, -0.389386385679245, -0.4152100086212158, 1.7233611345291138, -2.2830116748809814, 0.6759769916534424, -0.6598708033561707, 1.4331011772155762, 0.6981292963027954, 0.7152929902076721, -0.4038456678390503, 0.30179905891418457, 0.7316473722457886, -1.260722041130066, -0.2431911677122116, -0.7416719198226929, 0.3546009361743927, 1.3714677095413208, -1.3209035396575928, 2.259556531906128, -0.9249814748764038, -2.04215407371521, 0.2528333067893982, 0.4411042332649231, 0.9432043433189392, 0.39393818378448486, 0.6441637873649597, -0.7184860706329346, -0.16871118545532227, 0.786643922328949, 0.617855966091156, 0.6837126612663269], [-1.6061546802520752, -1.301137924194336, 0.3616291880607605, -0.2444252222776413, 2.526780843734741, 0.4754265248775482, -1.510581374168396, -0.5516060590744019, -0.36429789662361145, 0.014659176580607891, 0.5957601070404053, 0.4641515910625458, 0.4409126341342926, -0.5947904586791992, -0.16961883008480072, 0.04781734570860863, 0.7492239475250244, -1.3045473098754883, 1.2664093971252441, -0.9416089653968811, 0.1111319288611412, 0.5052143931388855, -0.37027621269226074, -1.5002822875976562, -0.44894739985466003, 0.2893090844154358, 0.13002179563045502, -0.3407253921031952, -0.5152269601821899, 0.7455801367759705, 0.03928125277161598, 0.7135137915611267, -0.8598344326019287, -0.7227223515510559, -0.7105805277824402, -0.024995049461722374, -0.1470394879579544, 0.22549192607402802, 0.113780677318573, -1.4626400470733643, 0.9066266417503357, 0.01875951699912548, -1.1061779260635376, -2.225273609161377, 1.4325408935546875, 0.7634931802749634, -0.48170650005340576, -0.43032947182655334, 0.07354909181594849, 0.9086906313896179, 2.1446280479431152, -0.6734294891357422, -0.9355512857437134, -0.07557535916566849, -0.7621660232543945, -0.5058082938194275, -2.025322914123535, 0.46572408080101013, -1.6446863412857056, -0.7886863946914673, -0.0032609535846859217, 1.4768152236938477, -2.000020742416382, 0.672132134437561], [-0.14724436402320862, 0.6658431887626648, 2.3774046897888184, 0.1355152577161789, -0.08584103733301163, 0.09703440964221954, -0.6572864055633545, -0.7540358901023865, 0.9997968077659607, -0.6301112174987793, 1.1191726922988892, -0.6895870566368103, 1.4148122072219849, -0.5301111936569214, -1.8088023662567139, 0.11532475799322128, -0.1969706416130066, -1.1862728595733643, 1.0739120244979858, -1.4625263214111328, 1.6880170106887817, -0.42778193950653076, 0.8251497149467468, -0.5762145519256592, 0.1576455533504486, 0.8271750807762146, 1.4623411893844604, -0.9089822173118591, 0.4049718976020813, -1.0158241987228394, 0.37470516562461853, 0.843425989151001, 0.6228086352348328, -1.362217664718628, 0.20922192931175232, 0.5227031111717224, 0.12531901895999908, 1.3472031354904175, 1.21578848361969, -1.2540555000305176, 0.6897420883178711, 0.7539142370223999, -0.771745502948761, -1.779715657234192, -1.4452558755874634, 0.6345988512039185, -0.6639096736907959, 2.2055249214172363, -0.9624272584915161, 0.6141199469566345, 0.6132838726043701, 0.44235360622406006, 0.47633445262908936, 0.860651433467865, -0.8254655003547668, 0.0126802958548069, -1.6908235549926758, 0.42997244000434875, 1.1003221273422241, -0.30943018198013306, -1.7584961652755737, 0.23029237985610962, -0.2993411719799042, 0.693252682685852], [0.304094135761261, -0.9795406460762024, 0.8548213243484497, -0.26539817452430725, -1.3111586570739746, 0.6515570878982544, -0.6535537838935852, 1.3899788856506348, 1.168984055519104, 1.1403709650039673, 1.4435973167419434, 0.539043128490448, -0.9498780965805054, 0.38499191403388977, 1.0895717144012451, 0.8919786810874939, 0.5974328517913818, 1.0686134099960327, -0.9956422448158264, -0.7392775416374207, 0.23446844518184662, -0.77448970079422, 1.742603063583374, -0.06682552397251129, -1.247488021850586, -0.3968566954135895, 0.6712383031845093, 0.023686323314905167, -1.1911760568618774, 0.10619322955608368, -0.46547022461891174, -1.169385313987732, -0.0686667338013649, -1.3271623849868774, -0.933266282081604, -1.6494945287704468, -0.25052589178085327, -0.3032767176628113, 0.8951970934867859, -1.010321021080017, 0.497372031211853, -0.9368414878845215, 0.7254644632339478, 0.366363525390625, 0.4684714078903198, -0.2601604461669922, 0.3502576947212219, -2.1380746364593506, -1.3322995901107788, 1.344861626625061, -0.6103225946426392, 0.0020254431292414665, 1.181409478187561, 1.447503924369812, 0.25313636660575867, -1.8674290180206299, 0.7656710743904114, -0.11673435568809509, -0.3800036907196045, 1.8374203443527222, 0.4526686668395996, 0.6459456086158752, -2.010944128036499, -0.5281452536582947], [0.8578203320503235, 0.2433617264032364, -0.40130850672721863, 1.2192282676696777, 0.17502781748771667, -0.9881114959716797, -0.04834659770131111, -0.6157063245773315, 1.0440242290496826, 0.6424925327301025, -0.38634341955184937, -0.23975403606891632, -2.0873842239379883, -1.1118642091751099, -1.6898179054260254, -0.4597649872303009, 0.33804258704185486, -2.1008410453796387, -1.1589080095291138, 0.9721843004226685, -1.8424568176269531, 2.318943738937378, -0.5668392181396484, 0.5924491286277771, 0.8553048968315125, 0.548673689365387, 1.7183029651641846, -0.9630798101425171, -0.1467503160238266, -0.7745380401611328, 0.6005368232727051, -0.46381404995918274, 1.3144433498382568, 0.6167640089988708, 0.7491637468338013, 0.8577916026115417, -0.41599270701408386, -0.7180396318435669, 0.17746834456920624, -0.5635838508605957, -1.0281175374984741, -0.10023507475852966, -0.017701277509331703, -1.7126753330230713, 1.2701910734176636, 0.11178485304117203, -0.8810380697250366, -0.336390882730484, 0.5305615663528442, 1.1256481409072876, -0.7773864269256592, -0.7090023756027222, -0.2576626241207123, 0.037717003375291824, 1.1999104022979736, 1.1262824535369873, -0.6019032597541809, -0.04025834798812866, -0.9869098663330078, -1.4583182334899902, 0.014036286622285843, -0.22329849004745483, -0.5843586921691895, -0.8887853622436523], [0.18473899364471436, 0.21712103486061096, 1.6215078830718994, -0.6755454540252686, -0.6012771129608154, -0.41007643938064575, 4.3176188468933105, 0.8811036348342896, -1.8522803783416748, 0.23037676513195038, -0.3997696340084076, -2.034473419189453, -0.9305092096328735, 0.09091866761445999, 1.9976834058761597, 0.39077454805374146, 1.0734220743179321, -1.8187286853790283, -0.26888978481292725, -0.5704087018966675, 1.5563158988952637, -0.4228500425815582, 0.26306894421577454, -0.6080401539802551, -0.11197277903556824, -1.70982027053833, 1.102053165435791, -0.8649424314498901, 0.034683939069509506, -0.952636182308197, 0.29797545075416565, 0.008841480128467083, 0.38414955139160156, -0.9387180209159851, 0.20762355625629425, 2.3342509269714355, 0.4469888508319855, 0.6128399968147278, -0.30611056089401245, -0.8507452607154846, 0.7586404085159302, -0.654950737953186, -1.1501998901367188, -1.32452392578125, -1.6608500480651855, -0.9140428900718689, -0.8723201155662537, 1.3134987354278564, 0.4700702428817749, 0.5082067251205444, 0.6553757190704346, -0.7530469298362732, -0.6220251321792603, 0.03174066171050072, -0.8677511215209961, 0.5308386087417603, -0.4150020182132721, -0.24551373720169067, -1.3524552583694458, -0.17061682045459747, -2.178892135620117, 1.408915638923645, -0.32078152894973755, -0.20066674053668976], [0.013480216264724731, 0.5423849821090698, -2.4061169624328613, -1.3796257972717285, 1.0393245220184326, 0.9543314576148987, -0.4059500992298126, -0.5317410230636597, -1.0990910530090332, 0.05114596337080002, -0.6776437759399414, 0.451871395111084, 0.7485758662223816, 0.06347616761922836, 0.0895310640335083, -2.942033052444458, -0.8733150959014893, -0.5090333819389343, 0.1432446837425232, 0.4143916368484497, -0.20450222492218018, 0.45750781893730164, 0.05031505227088928, -0.7946294546127319, 0.5427929162979126, -1.0436229705810547, -0.6055218577384949, -0.024660365656018257, 1.5126783847808838, 1.0502609014511108, -0.4007876217365265, 0.13384892046451569, 0.701612651348114, -1.8873438835144043, -0.2251710146665573, -1.1818772554397583, -0.589058518409729, 2.0858020782470703, -1.1231005191802979, -1.139402985572815, 0.9596447348594666, 0.7320978045463562, 0.7579843401908875, 0.26677483320236206, 1.1665242910385132, 0.08126173913478851, -1.1369777917861938, -1.0045990943908691, 0.45504769682884216, 0.1889941394329071, -0.11760120093822479, -0.31660300493240356, -0.33271822333335876, -0.5077804327011108, -0.5376747846603394, -1.2276639938354492, 0.7053084373474121, -0.3689940273761749, 0.8411840200424194, -0.8808578848838806, -1.1140801906585693, 0.45686590671539307, -0.19815459847450256, 0.45178642868995667], [-0.6162403225898743, -0.009755074046552181, 0.541056215763092, -0.11375758796930313, 1.626635193824768, 0.2526305615901947, 0.37570810317993164, 0.12343091517686844, -1.7779451608657837, 0.4256916046142578, -0.9478769898414612, -0.5269264578819275, -0.2630175054073334, -0.9614577293395996, 1.6846457719802856, 1.7362456321716309, 0.03076254203915596, -0.28804439306259155, -0.6584044098854065, -0.8560634255409241, 1.3760217428207397, 0.734710693359375, 0.919152557849884, 0.018755581229925156, -2.3845958709716797, 1.0552808046340942, -0.3155837953090668, 0.24634912610054016, -1.222182273864746, 0.08835319429636002, -0.39377889037132263, 1.2896084785461426, 0.43637093901634216, -0.34253379702568054, 0.11346820741891861, 0.34405606985092163, -0.19823379814624786, -0.2213340848684311, 0.08537189662456512, 0.49097222089767456, -2.2153568267822266, 0.49169620871543884, -0.7917382717132568, 0.29369693994522095, 0.744208037853241, -0.7000375390052795, 0.9017828106880188, -0.03780949115753174, 0.06997943669557571, -0.23149748146533966, -0.2799910008907318, -1.1319940090179443, 0.2628500759601593, -1.2082328796386719, -1.2107088565826416, 1.2496848106384277, 0.9451954960823059, 2.3724093437194824, -1.1201891899108887, -0.7774192690849304, 1.2774858474731445, -1.680464267730713, 1.4500541687011719, 1.1040395498275757], [0.46769335865974426, -0.07286491990089417, 0.050376810133457184, -0.6395597457885742, 0.295460045337677, 0.17324812710285187, 0.27731069922447205, -0.03266622871160507, 0.7591243386268616, 0.4218601584434509, -1.0482107400894165, -0.9913359880447388, -0.296196311712265, -0.6428263187408447, -0.45707938075065613, 0.9178079962730408, 2.080876350402832, 1.0895395278930664, -1.9488067626953125, 0.14559538662433624, 0.26861947774887085, 0.8449367880821228, 0.7754440307617188, 0.35056012868881226, -0.04179834946990013, 1.5596439838409424, 1.4925655126571655, 1.130256175994873, -0.08516617119312286, 1.2302500009536743, -0.8708228468894958, 0.3245508074760437, 1.1705851554870605, 0.34974291920661926, -1.006242036819458, -0.9465977549552917, -1.3830938339233398, -0.7794742584228516, 0.0904177874326706, 1.527230978012085, 0.18806935846805573, 0.14740724861621857, 1.8804678916931152, 0.43953096866607666, -0.5524875521659851, -0.04557519778609276, -0.3931101858615875, -1.1174044609069824, 0.34539151191711426, -1.048828363418579, 0.39657822251319885, 1.0584913492202759, -0.4385138750076294, -0.8004763722419739, -0.047264788299798965, -0.55733323097229, 0.059241242706775665, 0.6312457323074341, 0.5183071494102478, 0.19642391800880432, -1.2430214881896973, 0.4918675422668457, -1.5241944789886475, 0.03124268166720867], [1.365648865699768, -0.7337307333946228, -1.155158281326294, -2.4743800163269043, 1.8121834993362427, 0.49789583683013916, -1.8566408157348633, 1.7119625806808472, 0.5336116552352905, -0.758029580116272, 0.7067430019378662, -1.0528618097305298, -1.464113712310791, -0.4388803541660309, 0.05223781242966652, 0.9106728434562683, -0.16295126080513, 0.16040560603141785, 0.04043335095047951, -0.09465735405683517, 1.0954338312149048, -0.08386148512363434, -0.8703456521034241, 0.2684330642223358, 0.10884921997785568, 0.3533822298049927, -0.1986943930387497, -0.07450634241104126, 0.6882397532463074, 0.9913434982299805, -1.007788896560669, -1.5720502138137817, -0.6899292469024658, -0.45346856117248535, -1.1438698768615723, -0.9240174293518066, 0.25320297479629517, -0.21761803328990936, -0.06764394044876099, -1.7556968927383423, -1.006245732307434, -0.9182400107383728, 0.0062758647836744785, 1.3870760202407837, -2.0808844566345215, 0.8380023837089539, -1.4158321619033813, -0.23911917209625244, 0.4120502471923828, -1.571972131729126, -0.9493461847305298, 2.144397735595703, 0.39865899085998535, 0.5683072805404663, -0.787668764591217, -0.4713278114795685, -0.34249061346054077, -0.5837249755859375, -2.1691296100616455, 1.8188517093658447, 0.577427089214325, -0.06506083160638809, -2.2581872940063477, 0.5731312036514282], [-0.0863056555390358, -0.22974757850170135, 0.599014163017273, -0.2601131200790405, -0.35103631019592285, 0.6442437171936035, 0.2617078721523285, 1.3919322490692139, -0.8953238129615784, -1.0627644062042236, 0.5730509161949158, -0.7540618181228638, 0.5820738077163696, 1.800520658493042, 0.9065081477165222, -0.7365075349807739, -0.5156030058860779, -0.2893209755420685, -0.49151402711868286, -0.1049751341342926, 0.1862112432718277, -0.07016107439994812, 0.08662951737642288, -1.1716465950012207, 2.1500203609466553, -0.21875175833702087, -1.6489304304122925, -1.1452815532684326, -0.6897614598274231, -0.8554872870445251, -0.7606265544891357, -0.030542809516191483, 0.23733051121234894, 1.5629457235336304, -1.4929178953170776, -1.5241683721542358, -0.01934373565018177, -1.097913146018982, -0.6796526908874512, -1.6757540702819824, -0.22355620563030243, 2.4116880893707275, 0.658066987991333, -0.6361907124519348, 1.9037448167800903, -2.1248695850372314, -1.505888819694519, -0.33924633264541626, -0.7419872283935547, 0.09166046231985092, -0.04697352275252342, -1.538138508796692, -1.24580979347229, -1.4496251344680786, -0.9433232545852661, 0.6002871990203857, -0.9918553233146667, -0.5697107911109924, 0.44680535793304443, -0.985660970211029, 1.8253098726272583, -0.29080072045326233, -0.581657350063324, 0.7843846678733826], [-0.19351309537887573, 1.276002049446106, 1.6990936994552612, 0.42475995421409607, 1.512441635131836, 1.4787112474441528, -0.42269662022590637, 2.308346748352051, -0.25219205021858215, 0.8841778635978699, -0.3282042443752289, -0.24407783150672913, -0.02025046944618225, -0.06402692943811417, 0.09213783591985703, 0.9958000779151917, 0.5096185207366943, -1.9057645797729492, 0.07929737120866776, -0.3379098176956177, 1.0640813112258911, -0.38283857703208923, 0.19509129226207733, -0.4287341833114624, -0.12363462150096893, 1.4501224756240845, 0.2986154556274414, 1.0320696830749512, 0.1626143604516983, -0.9449169039726257, 0.3890089988708496, -1.9258830547332764, 0.856637716293335, -0.46362656354904175, -2.3254239559173584, 0.35634633898735046, 1.8963720798492432, -0.15547339618206024, -0.10783270001411438, 1.7145168781280518, 0.03517425060272217, -0.17356298863887787, -0.1625448763370514, 0.6192856431007385, -0.07568210363388062, -0.891448974609375, -0.47752606868743896, -0.39728862047195435, 0.9974278807640076, -0.10394632071256638, -0.3978472948074341, 1.481375813484192, 1.0682542324066162, 0.5654791593551636, 0.3094862103462219, -1.0970298051834106, 1.312962293624878, 0.10041960328817368, 0.13325753808021545, -0.8228005170822144, -0.3338324725627899, 1.4389688968658447, 0.027228524908423424, 0.33258122205734253], [-1.3279781341552734, 1.8225815296173096, 0.48354220390319824, 1.6496448516845703, -1.9701186418533325, -0.3515259921550751, -1.629658818244934, 2.2647488117218018, 1.7833832502365112, -2.1123218536376953, -0.18971161544322968, -1.1520851850509644, -0.1787775456905365, 0.48321136832237244, -0.39415377378463745, 1.119091510772705, -1.5828373432159424, -0.09900327771902084, 0.6356881856918335, -0.8855932950973511, 0.48356473445892334, 0.28086861968040466, -0.6369174122810364, -1.932342529296875, -0.2802661061286926, -0.12959399819374084, -0.8502345085144043, 1.0276808738708496, 0.006990349851548672, 0.4863191246986389, -1.027069330215454, 0.14094947278499603, 0.6927112936973572, -0.7391475439071655, -0.4597601890563965, 0.8210856318473816, 1.6463985443115234, 1.2536182403564453, 0.10553839057683945, -0.5952879786491394, 0.08386775106191635, -0.8887174725532532, 1.1998660564422607, -1.1609777212142944, 0.16656160354614258, -1.2164722681045532, 0.372478723526001, 0.022901451215147972, 0.3719198405742645, -0.7877941727638245, 0.2503448724746704, 1.5126231908798218, -0.2673235535621643, -1.5809060335159302, 0.11441277712583542, 1.6333556175231934, 1.4689465761184692, -1.0922714471817017, -0.32982856035232544, -0.6501445770263672, 0.6656296849250793, 0.34216445684432983, -0.4511238932609558, -0.16386544704437256], [0.2991338074207306, -0.7362145781517029, -0.26730507612228394, 2.2563390731811523, 0.19286799430847168, -0.379057914018631, 0.5512635707855225, 0.3605673313140869, -0.04579322412610054, 0.7864142656326294, -0.3230815529823303, 0.83531254529953, 1.0212827920913696, 0.40914443135261536, 2.3275794982910156, -0.047487203031778336, 0.6228352189064026, 0.37318602204322815, 0.7548627853393555, 0.8689396381378174, 1.1875700950622559, 0.3912874162197113, 0.41568347811698914, 0.4245399534702301, -0.033966004848480225, -0.295747309923172, -0.8021383881568909, -0.42701947689056396, 0.210132896900177, -0.1738511174917221, -1.0629408359527588, 0.8949649333953857, -1.2098337411880493, -0.27213576436042786, -1.6322917938232422, -1.2716670036315918, 0.7430375814437866, -0.9538534879684448, -0.9844955205917358, -1.2480387687683105, -2.616562604904175, -1.681438684463501, -0.028332499787211418, -1.358909249305725, -1.3771744966506958, -0.20580331981182098, 0.17994195222854614, 1.193532109260559, 0.8078858852386475, -0.7141728401184082, -1.5540931224822998, 0.36890485882759094, 0.16327665746212006, 1.3992881774902344, -1.4570444822311401, -1.5400689840316772, -0.7120983004570007, -0.45626047253608704, 0.13963067531585693, -0.8389446139335632, -1.2481286525726318, 0.23784689605236053, 0.2671297788619995, 0.9946768283843994], [-1.863208532333374, 2.2089991569519043, -0.008378379046916962, -0.7001362442970276, 0.25543636083602905, -0.8587783575057983, 1.6385748386383057, -0.15909415483474731, -0.02862037532031536, 0.6179154515266418, -0.14398734271526337, -0.1858648806810379, -0.7545278668403625, -0.2298075407743454, 0.04668644443154335, -0.4367406368255615, -0.12126800417900085, 1.1629621982574463, -0.889452338218689, 0.5484063029289246, 0.8376567363739014, 0.642341136932373, -0.799897313117981, 1.0617574453353882, -0.027947667986154556, -0.7610159516334534, 0.2649350166320801, 0.9073662757873535, -1.295651912689209, 0.6449421048164368, -0.03791290894150734, 1.1513231992721558, 0.7334784865379333, -0.4003812372684479, 0.7592065930366516, 0.2916790544986725, 0.9555603265762329, 0.5128862261772156, -0.6946528553962708, -0.08648417145013809, -1.2619941234588623, -1.1954271793365479, 0.5819422602653503, 1.2950412034988403, 1.5511653423309326, -1.5030306577682495, -1.6385459899902344, -1.2755513191223145, -0.6050988435745239, -0.5762532353401184, -1.0134505033493042, 0.22106286883354187, 0.37303996086120605, 0.09642722457647324, 1.236965298652649, 0.7071118354797363, -0.34436526894569397, 0.6593518853187561, -0.6826441287994385, 0.7907326817512512, -0.42128559947013855, 0.8452625870704651, -2.5112428665161133, 0.2231922149658203], [-0.24165473878383636, 1.359998345375061, -1.8810443878173828, 0.6210938692092896, -1.5687576532363892, -0.38679397106170654, 0.0761483758687973, 1.0467904806137085, -0.5546861886978149, 2.173297643661499, 1.0698868036270142, -1.553985834121704, 0.4698963463306427, -0.09808029979467392, -0.2989963889122009, 0.018124597147107124, -0.17033228278160095, 0.6203957796096802, -1.1836886405944824, 1.1117392778396606, -0.11464785784482956, 0.8617870211601257, 0.8130624294281006, -1.4465125799179077, -0.2404296100139618, 1.0904239416122437, -0.05464261770248413, -0.35010606050491333, -0.6041091680526733, 0.49087631702423096, -0.3348093330860138, 1.5293809175491333, -0.11988811939954758, -1.016972541809082, -1.339469075202942, 0.01795841194689274, -0.3151320219039917, -0.0019257526146247983, -1.228602409362793, -0.5169420838356018, -0.18341957032680511, 2.7134790420532227, -0.42382878065109253, -0.17847619950771332, 0.7452274560928345, -1.192987084388733, 0.19979815185070038, -1.1514099836349487, -0.22954495251178741, 0.5847936272621155, 0.04485771432518959, 1.0629184246063232, 0.5683735013008118, 0.9310590028762817, 1.0107783079147339, -1.0584794282913208, 0.10968927294015884, 0.6389360427856445, -0.1886848360300064, 0.7247204780578613, 0.6149165630340576, -0.6286052465438843, 0.08672334998846054, 0.35872623324394226], [-0.06504286080598831, -1.8155791759490967, -0.07499909400939941, -0.48880642652511597, -0.744949221611023, 0.533686101436615, 0.2022937536239624, -0.675595760345459, 0.5748748183250427, 1.3742380142211914, 2.8134751319885254, -1.3664321899414062, -0.38335415720939636, 1.8625471591949463, -0.0908200815320015, -0.5090672969818115, -0.3109760582447052, -0.9877999424934387, -0.06595880538225174, -0.9141236543655396, -0.48651444911956787, -1.2798022031784058, -1.9533213376998901, -0.7103608250617981, -0.6921827793121338, 0.9694247245788574, -0.48821526765823364, -1.7250382900238037, -0.34911319613456726, 2.3603408336639404, -1.3159892559051514, -0.32806289196014404, 1.0604097843170166, -0.15324202179908752, -0.0671919584274292, 0.6204003691673279, 1.4044522047042847, 0.22966518998146057, -0.19995099306106567, -0.3393830358982086, 0.4823882281780243, 1.3803695440292358, -1.0858818292617798, -0.6121743321418762, -1.4627811908721924, -1.5570694208145142, -1.780203938484192, -0.35806581377983093, -0.32747164368629456, -0.8170402646064758, 0.9599445462226868, -1.1545389890670776, 0.32565397024154663, 0.05288508161902428, 1.02946937084198, -1.638195514678955, 2.373903274536133, -0.8942908644676208, -0.6543784737586975, 0.2788761258125305, -1.4863897562026978, -1.8779650926589966, 0.7065390348434448, -0.2268889993429184], [-1.4802937507629395, 0.2767046391963959, -0.6373761296272278, -0.1773620843887329, 0.06392372399568558, 0.28134408593177795, 1.3333444595336914, 0.30952486395835876, -0.6680153012275696, -0.438258558511734, -0.6963364481925964, -1.2352122068405151, -0.15390872955322266, -0.4873707890510559, 0.5721396803855896, -0.13453496992588043, 0.34358811378479004, -0.06944721192121506, -0.7841594815254211, 1.2074283361434937, 0.7633761763572693, 0.08075142651796341, 0.5356301665306091, 0.9676823019981384, 1.0548982620239258, -0.5084587931632996, -0.8824874758720398, 1.4249337911605835, 0.16509069502353668, -1.3583744764328003, 1.4742774963378906, -0.8356069922447205, -0.7137624025344849, -2.693939685821533, -0.3731021583080292, 0.018079647794365883, 1.2049131393432617, -0.9494905471801758, 1.220090627670288, 0.1488293707370758, -0.1623048484325409, -1.7923812866210938, 1.7132415771484375, -0.14243851602077484, -1.5968824625015259, -1.9164361953735352, 0.8476517200469971, -0.7170501947402954, 0.07676972448825836, 0.46336492896080017, 0.1959104686975479, -0.2695120871067047, -0.6285055875778198, 0.592466413974762, -0.478620707988739, -0.10421216487884521, -0.8164476752281189, 0.48650556802749634, 0.5461683869361877, 0.3940945565700531, 2.027437925338745, -1.781252145767212, 0.8541333675384521, 1.171244502067566], [0.4962822496891022, -1.7565377950668335, -0.0756511241197586, -0.6033298969268799, -0.425028920173645, -2.5934665203094482, 0.9844785928726196, -0.47457754611968994, 0.6338001489639282, 1.155678391456604, 0.9951296448707581, 1.4917136430740356, 0.0691644549369812, 0.22325772047042847, -0.26647675037384033, -0.3667800724506378, -0.4604889750480652, 1.13200843334198, -0.1888808012008667, 0.18068033456802368, -1.0690778493881226, -0.5458657741546631, 0.5546016097068787, 0.3900952637195587, 0.10697490721940994, -0.8132901191711426, -0.5187330842018127, 1.2545015811920166, 0.389077752828598, -0.5597866773605347, -1.4065029621124268, -0.08964349329471588, -0.7778041958808899, -0.3123989701271057, -1.174927830696106, 0.21638140082359314, 0.3268990218639374, -0.1371685117483139, 0.8599926233291626, 0.18583858013153076, -0.5756564140319824, -2.218867063522339, 1.4233707189559937, -0.443239688873291, -0.9723198413848877, -0.05346839874982834, -0.01035089697688818, 0.5242742896080017, -0.213079571723938, -0.19943562150001526, -0.8486926555633545, 0.47767797112464905, -0.9048831462860107, 1.1910367012023926, 0.6852657198905945, -0.9041643738746643, 1.4478915929794312, 0.6815788149833679, 1.7796121835708618, -0.2074926495552063, 0.7969545722007751, -0.5153307318687439, -0.1981441080570221, 1.6300441026687622], [0.6528186202049255, 1.6498686075210571, -2.7774500846862793, 0.5200836658477783, -0.23950505256652832, -1.2596553564071655, 1.1025958061218262, 1.6112985610961914, 1.0243808031082153, 1.4403048753738403, -0.47308143973350525, -2.5442166328430176, -0.0633741021156311, -1.161375880241394, 2.0875043869018555, -0.4368046224117279, 0.1633685827255249, 0.04959508404135704, -0.517916202545166, 0.32969993352890015, -1.3146682977676392, 0.4693933129310608, 0.3417360782623291, -1.7197481393814087, 0.534290075302124, -2.2437052726745605, -0.26050612330436707, -0.328297883272171, 0.23051388561725616, -0.21218976378440857, 0.05748111754655838, 0.4916835129261017, 0.10401757806539536, 1.0090612173080444, -1.1180610656738281, -0.04625799134373665, -0.6058371663093567, 0.168552964925766, 1.7947956323623657, 1.106699824333191, -0.9215479493141174, 0.9315454959869385, -0.3197440505027771, -1.302081823348999, 0.7733983993530273, 2.5947906970977783, -1.5043416023254395, 1.6652135848999023, -0.9890479445457458, -0.7383314371109009, -0.5509803891181946, -0.001048246631398797, -0.34572935104370117, 0.6900564432144165, 1.3278571367263794, -0.6673586964607239, -1.042373538017273, -0.6064855456352234, 0.17446625232696533, 0.27511781454086304, -1.2500460147857666, -0.0050910660065710545, 0.3672778606414795, -1.9106866121292114], [-1.4954525232315063, 0.06593883037567139, 2.379452705383301, 1.1123751401901245, -0.3531401753425598, -0.47942817211151123, 0.2724882662296295, 1.6418498754501343, 2.2153961658477783, -2.49812650680542, -0.6251581907272339, 0.003415327286347747, -0.06296831369400024, -1.0825786590576172, -0.7691473960876465, 0.3574278950691223, -0.7685134410858154, 0.6520370244979858, -0.378438264131546, -1.3742069005966187, 0.9818444848060608, 0.8154810667037964, 0.0003695568593684584, 1.010206699371338, 0.9696879982948303, -0.5948877334594727, 1.5430207252502441, -1.9932210445404053, 0.8289650678634644, -0.11233287304639816, -0.34272339940071106, -1.172389030456543, 0.3142216205596924, -0.6144468784332275, -0.08234991133213043, -1.3244026899337769, 0.9122462868690491, 0.4099728763103485, 0.09864210337400436, -1.5346375703811646, 1.2337651252746582, -0.3101809024810791, -1.363532543182373, 1.0433508157730103, -0.6358967423439026, -0.26452258229255676, 1.0221439599990845, 0.8566275238990784, -0.5479128956794739, -0.41539573669433594, -0.79615318775177, 0.9456530809402466, 0.5251405239105225, -0.6924599409103394, 1.9619089365005493, -0.495528906583786, -0.36703774333000183, -0.4528505504131317, 0.8537227511405945, -1.3861886262893677, 1.0406908988952637, 0.8512740135192871, 0.31263965368270874, -0.14624032378196716], [-0.14347094297409058, 0.5768842101097107, -3.1224687099456787, -0.44060584902763367, 0.8232318162918091, 0.058089688420295715, -0.14573106169700623, 0.32549551129341125, -0.25081557035446167, -0.23824258148670197, 0.6025396585464478, 1.7434380054473877, -1.431766390800476, 0.9812753200531006, -0.9612783193588257, 0.2922498285770416, 0.06399930268526077, 0.7821898460388184, 0.5621603727340698, -0.282591313123703, 2.3897573947906494, 0.406474769115448, 0.6855259537696838, -1.5682767629623413, -0.49217548966407776, 2.388477087020874, -0.44300827383995056, 0.8826018571853638, 0.24726304411888123, 1.2015308141708374, -1.4147592782974243, -1.7386263608932495, -0.028935037553310394, -0.7692374587059021, -0.4719488024711609, -0.19577765464782715, -0.2598515450954437, 0.36993348598480225, 0.019015055149793625, -0.18742789328098297, 0.2607751190662384, 1.8314063549041748, 0.08033192902803421, 0.21270857751369476, -0.725945234298706, 0.9592713713645935, -0.24443888664245605, -0.2187582403421402, -0.34694814682006836, 1.3318947553634644, 2.07737135887146, 2.0077905654907227, -0.0991801992058754, 1.3957008123397827, 0.13663911819458008, 0.3694712221622467, -0.18728791177272797, -0.6065331697463989, 0.578121542930603, -0.9060246348381042, -1.636027455329895, 0.7563326358795166, 0.6619620323181152, 1.18271005153656], [0.554259181022644, 1.5677897930145264, 0.18978706002235413, 0.47583892941474915, -0.08480995893478394, -0.5119505524635315, -0.16667887568473816, 0.8525910377502441, -0.4143937826156616, 0.48790696263313293, 0.9659739136695862, -0.03846121206879616, 0.7181660532951355, 1.3211166858673096, 2.524704933166504, -0.05393052473664284, -0.8281831741333008, -1.564539909362793, 0.7317196130752563, 0.19693320989608765, 0.5649603605270386, 0.9679840207099915, 0.7055966258049011, 1.8004192113876343, -0.43118563294410706, 0.9516292810440063, 1.3095011711120605, 1.3488529920578003, -0.31075188517570496, 0.032673973590135574, 0.016618400812149048, -2.2131080627441406, -1.0809924602508545, 1.0589282512664795, -0.6033554673194885, 0.6341848373413086, -1.0403106212615967, 0.40447428822517395, -0.3052501976490021, -0.07354938983917236, 0.7413758635520935, -0.6614853739738464, -1.7176822423934937, 1.7083078622817993, -0.4237706661224365, 0.6369358897209167, -0.3209122121334076, -2.1983604431152344, -0.27199065685272217, 1.907964825630188, -0.8473294377326965, 0.39414066076278687, 0.8191450834274292, 0.34176772832870483, -1.1885405778884888, 1.338153600692749, 0.2940283715724945, -0.508440375328064, 0.3954222798347473, -1.086134910583496, 1.5561379194259644, 0.709577202796936, -1.7786719799041748, -0.5240187048912048], [0.8310648798942566, -0.3646482229232788, 1.1324931383132935, 0.5672031044960022, 0.17199836671352386, -1.6563472747802734, 0.4361325800418854, 0.28899016976356506, 0.8043091893196106, 0.006138606928288937, -0.4517522156238556, -1.479624629020691, -0.524643063545227, 2.055896759033203, -1.6739308834075928, 1.4092228412628174, 1.6269121170043945, 0.7864728569984436, -1.6471530199050903, 0.303264856338501, -1.9158488512039185, -1.5177942514419556, 0.4932246804237366, -1.1991095542907715, 0.6252381205558777, 0.9043100476264954, -1.1461389064788818, 0.6235209703445435, 2.159588575363159, -0.12551085650920868, -0.49412789940834045, 0.19985784590244293, -0.7146630883216858, -0.522244930267334, 1.3831475973129272, -0.09044443070888519, 0.049347471445798874, 0.41591471433639526, -0.007576784584671259, -1.5150760412216187, -1.1821062564849854, -1.788718581199646, 0.9679597020149231, 0.4737842381000519, -1.57156240940094, 1.0340633392333984, 0.18590088188648224, 0.8597253561019897, 0.2753320038318634, -0.196528822183609, -0.18296858668327332, -0.9545050859451294, -0.6010130643844604, 2.620831251144409, 0.1702696830034256, 1.5836718082427979, 0.8640979528427124, 1.2996182441711426, -1.5429354906082153, -0.24532845616340637, 0.3640824258327484, -0.8640836477279663, 1.4105464220046997, 0.09218741208314896], [-0.29444169998168945, 0.22756874561309814, 0.39069080352783203, 0.21660780906677246, -1.8861936330795288, 0.08517859131097794, 0.9841690063476562, 0.7925792336463928, -0.46744441986083984, 0.7698493599891663, 0.25767216086387634, 0.5456740260124207, 1.0473878383636475, -0.4706416428089142, -0.39062321186065674, -0.7826898097991943, -0.24610339105129242, 0.06563431769609451, -0.03936086967587471, -0.9533559083938599, 0.22806517779827118, 0.20417776703834534, 0.12964734435081482, -0.42015594244003296, 0.11445777863264084, 0.8126932382583618, -1.567688226699829, -0.33888980746269226, 0.36993154883384705, 0.4708409309387207, 1.4327342510223389, -0.14238502085208893, -0.520655632019043, -1.2599520683288574, 0.7130008339881897, -0.41729044914245605, 1.2229900360107422, 1.212717056274414, -0.19034799933433533, 0.4951568841934204, 0.8259795904159546, -1.1470775604248047, 0.5225159525871277, 1.1872596740722656, -0.3881753087043762, -1.7147270441055298, -0.7578975558280945, -0.5287710428237915, -0.4245662987232208, -0.2777819037437439, -1.1821109056472778, -0.11085081100463867, 1.3282090425491333, 0.8947414755821228, -1.3934576511383057, 0.38240382075309753, -0.23292359709739685, -0.8934667706489563, -0.6930166482925415, -0.8203148245811462, 1.5352630615234375, 1.0365082025527954, -1.1559278964996338, -1.6189488172531128], [-1.6367164850234985, 1.7359713315963745, 1.9776020050048828, -0.6299989223480225, -1.4391727447509766, -0.4408565163612366, -0.18088500201702118, -0.7980747222900391, 0.5164413452148438, 0.3564543128013611, 0.25554341077804565, -1.6304658651351929, 2.4694302082061768, -0.9652957916259766, 0.029760215431451797, 1.6009365320205688, -1.8757503032684326, -0.4950712323188782, 1.6737431287765503, -0.6757602691650391, 1.3706004619598389, 0.7742315530776978, -0.5552089214324951, -0.8151543140411377, -0.18568375706672668, 0.23156170547008514, -0.487112432718277, -2.11370587348938, -0.17937883734703064, -2.4634265899658203, -0.3888433873653412, -0.8360457420349121, 0.09215601533651352, -0.6015892028808594, 0.7847490310668945, -1.5435724258422852, -0.0933111384510994, -0.5057604908943176, -0.6301407217979431, -1.288061261177063, 1.4854671955108643, -1.6538212299346924, 0.2626993954181671, 1.2089848518371582, 0.5903070569038391, -1.6126863956451416, -1.2244869470596313, -0.4149792492389679, 1.351866364479065, 0.39101916551589966, 0.8232529759407043, -0.9032483100891113, -0.8327010869979858, -2.0757153034210205, 1.511106252670288, -0.21571236848831177, 0.5874793529510498, 0.42055588960647583, 0.2807084918022156, 0.3276835083961487, 0.7782264351844788, -0.9116438031196594, 0.6063229441642761, -0.16220521926879883], [-0.936037003993988, -0.307976096868515, 0.10682521015405655, 0.5374422669410706, 1.2321280241012573, 0.07209748029708862, 0.3288699984550476, -0.48876136541366577, 1.370722770690918, 0.4980076551437378, 0.6536166667938232, -0.9435197710990906, 0.3992295563220978, -2.0020229816436768, 0.4013844430446625, 0.9333937764167786, 0.7385216951370239, 0.09606102854013443, -1.1652246713638306, -1.2510709762573242, -1.0599018335342407, -0.3336523473262787, -0.8338296413421631, -0.17891934514045715, -0.3833337724208832, -0.41240057349205017, 1.0281813144683838, -0.23442435264587402, 0.987955629825592, 2.1043541431427, 0.0013235623482614756, -0.2385065257549286, 0.7423384785652161, 0.042751044034957886, -0.3897445499897003, -1.1061439514160156, 0.6452707648277283, -0.6766804456710815, 0.1770104467868805, -0.18131619691848755, 0.4566626250743866, 0.39388346672058105, 1.5229130983352661, 1.2010211944580078, 0.18645954132080078, 0.497942179441452, -1.8329628705978394, 0.5015623569488525, -0.9155809283256531, 0.1686611771583557, 0.17852628231048584, -0.15719600021839142, 0.9718838930130005, 1.8899884223937988, 0.23616991937160492, -0.40657222270965576, 0.8277578353881836, -2.3875529766082764, -0.6886829733848572, 0.3052385449409485, 1.2654982805252075, -1.105748176574707, 0.3112216293811798, -0.4259926974773407], [0.24382410943508148, -0.3763616979122162, 0.4966067671775818, -0.12111825495958328, -2.3594772815704346, -0.16628167033195496, 0.3267776072025299, 0.35145536065101624, -0.19663965702056885, 1.6284894943237305, -1.362259030342102, 1.759156584739685, 0.5600895881652832, -0.9504475593566895, -0.06383451074361801, 0.1604573130607605, -2.226848840713501, -2.5148916244506836, 0.08439807593822479, 1.7902512550354004, -2.555436611175537, -0.7787429094314575, -1.8979144096374512, 0.2738983929157257, -0.744575023651123, 1.1536548137664795, -0.9477301239967346, 1.1342291831970215, -1.026449203491211, -1.7206804752349854, 1.231143832206726, -1.4543544054031372, 0.7754346132278442, -1.2715586423873901, 1.2194461822509766, 0.7765166163444519, 0.5641042590141296, 0.19030070304870605, -0.6464598178863525, 1.0518250465393066, 0.001653803512454033, 1.197226881980896, 0.3189104199409485, -0.18985751271247864, -0.8098820447921753, 0.27648788690567017, -0.4634716212749481, 0.9169362187385559, 0.019638068974018097, -1.268768310546875, -0.7647994756698608, 1.3440980911254883, -0.2842363715171814, -0.1194445863366127, 1.239658236503601, -0.17299756407737732, -1.1705241203308105, -0.7469181418418884, 0.5789840817451477, 0.625228762626648, -2.590557336807251, 0.343746542930603, -1.4860560894012451, 0.11318347603082657], [0.40440335869789124, -0.9372751712799072, 0.21494123339653015, -0.3473680913448334, 1.5771111249923706, -0.5547624230384827, -0.08479438722133636, 1.6983938217163086, -0.9539600610733032, -0.8122336864471436, -1.1556676626205444, 1.427957534790039, -0.5786747932434082, 1.2613041400909424, -1.2250272035598755, 0.8355156183242798, -2.4429144859313965, 0.08642707020044327, -0.8537917137145996, -2.2272863388061523, 0.18970684707164764, 0.009966288693249226, -0.27372264862060547, 0.366364449262619, 2.682584285736084, 1.2111706733703613, 0.5569418668746948, 0.014759618788957596, 0.05453743413090706, 1.2036820650100708, 1.607987880706787, -1.4338223934173584, -1.139721393585205, 0.6179025769233704, -1.3109029531478882, -0.08867378532886505, -0.26039692759513855, 0.24811385571956635, -1.5116186141967773, -0.6053169965744019, 1.4495787620544434, 0.5423167943954468, -0.21663042902946472, 1.1455535888671875, -1.6589645147323608, 0.1612941473722458, 0.934993326663971, 0.38926735520362854, -1.0133033990859985, 0.9723055958747864, -1.6421656608581543, -1.8912843465805054, -0.35753822326660156, -0.9856659770011902, -0.16664990782737732, -1.4519600868225098, -0.40405526757240295, 1.4667726755142212, -0.18453317880630493, -0.9156052470207214, -0.19208616018295288, 0.43572354316711426, 1.3190248012542725, 0.09036710113286972], [0.9807772040367126, 0.9067057371139526, -0.8883988261222839, -0.40180325508117676, -1.1683748960494995, 0.1535642147064209, -0.09177196770906448, -0.6639836430549622, -1.612586498260498, -0.12061956524848938, 1.1332916021347046, -0.01727345958352089, -0.24118104577064514, 0.29282596707344055, -1.6059879064559937, -1.1990505456924438, 0.11341080814599991, -0.44593578577041626, 0.8505385518074036, 0.9282820224761963, -0.5613819360733032, -0.07115485519170761, -0.44946590065956116, 0.308174729347229, 0.7667542099952698, 0.3405986726284027, -0.8537125587463379, -0.40056630969047546, -1.2470961809158325, 1.1253267526626587, -0.501767098903656, -0.23077847063541412, -0.1052340641617775, -0.46953150629997253, 0.9922837615013123, -1.5655078887939453, -1.1321579217910767, -0.5088231563568115, 0.25508779287338257, 0.12977291643619537, -0.20986950397491455, -0.060546986758708954, -0.09670433402061462, -1.0316681861877441, 2.147387742996216, -1.433976411819458, -1.6958279609680176, 0.35588544607162476, 0.2718682289123535, -1.0363972187042236, -0.4713763892650604, 1.765019178390503, -0.6113564372062683, -1.1601344347000122, -0.7688955068588257, -0.15232990682125092, 0.22164210677146912, 0.3888825476169586, 0.7266613245010376, 0.888988733291626, -1.1768642663955688, -0.5050640106201172, 0.21100939810276031, -1.7863692045211792], [1.9725099802017212, 1.566840410232544, 0.6521214246749878, 1.0498132705688477, 1.948317050933838, 0.10600874572992325, -0.39302197098731995, -1.5214424133300781, 0.9295640587806702, 1.6328656673431396, -0.43002578616142273, -1.4140493869781494, -0.9163156747817993, -0.28126582503318787, -2.1483328342437744, -0.8105567693710327, -0.0981508269906044, 0.15000198781490326, -0.603175699710846, -0.269187331199646, -1.3944575786590576, 0.2616419494152069, -0.005186745431274176, -0.4402238726615906, -1.731313943862915, 2.2078752517700195, -0.8223685026168823, 1.6521414518356323, -0.06394171714782715, -0.8138890266418457, -1.5349688529968262, 0.8142386078834534, -1.4840073585510254, 1.6356719732284546, -0.33627021312713623, 1.6300225257873535, 1.8047932386398315, 1.0656362771987915, -0.22050733864307404, 0.27542635798454285, -0.8848585486412048, -0.8654125928878784, -1.4274927377700806, 0.087853342294693, 0.6366265416145325, -0.5984776020050049, -0.30581068992614746, 1.1164063215255737, -1.0576658248901367, -2.195847511291504, -1.2644248008728027, -1.483986258506775, 0.8064257502555847, 0.43794727325439453, -1.4904688596725464, 0.8455149531364441, 0.8141880631446838, -1.1917637586593628, 1.7209980487823486, -3.01293683052063, -0.08033007383346558, -0.1474263221025467, -0.18566805124282837, -0.5760247111320496], [0.3900899291038513, 0.10548052191734314, 0.551301121711731, -0.12943868339061737, -0.6424511075019836, 0.7694863080978394, 0.8274678587913513, 0.4963567554950714, 0.14337922632694244, -0.48328763246536255, 1.2228087186813354, -0.11862718313932419, -1.0213035345077515, 0.08629342168569565, -0.5576165914535522, 0.1378907710313797, 1.324381709098816, 0.28712761402130127, -0.8271762728691101, -0.04664268717169762, 1.4815375804901123, -0.08341644704341888, -0.9954192042350769, 0.014052479527890682, 0.6530270576477051, -1.032019019126892, -1.013582468032837, -1.3964227437973022, 0.9729493260383606, 1.08818781375885, -1.2577427625656128, 1.7742464542388916, -0.7454267144203186, 1.2855018377304077, 1.2524585723876953, 0.7040137052536011, -1.0717976093292236, 0.0009499720181338489, 0.5375391840934753, -0.6218576431274414, -0.007260793820023537, -0.5751233100891113, -1.0841752290725708, 1.69052255153656, 1.4078710079193115, 1.905003547668457, 0.8413408398628235, -2.7376937866210938, -1.1313755512237549, 0.5175231099128723, -0.01392925065010786, -1.1310240030288696, -1.7076600790023804, 1.904354453086853, 0.3961026668548584, -0.5201712250709534, 1.7790865898132324, -0.3356243968009949, -0.2032841145992279, 1.124766230583191, -1.4160412549972534, -0.9645164012908936, -1.5946056842803955, -0.06277308613061905], [-0.6018970608711243, -0.6849216222763062, -0.17741349339485168, -2.2025818824768066, 0.26597028970718384, -0.01924501545727253, -1.4942139387130737, 0.9941176772117615, 0.34508687257766724, -1.135515570640564, 0.1788214147090912, -0.655802845954895, 0.7472876310348511, 0.469833642244339, -0.2878187596797943, 0.4731006324291229, -2.012805223464966, -0.7764520049095154, 1.2285948991775513, -2.0503790378570557, 0.4315643012523651, 0.754831075668335, 0.5533469319343567, -1.020632028579712, -1.0613594055175781, -0.6168928146362305, 0.9952303767204285, -0.820219874382019, -0.20737485587596893, -1.1010340452194214, -0.3943099081516266, -0.0769246444106102, -0.4231026768684387, -0.31667855381965637, 1.7035276889801025, 0.35238417983055115, -1.828608751296997, 0.17581771314144135, -0.36631229519844055, -1.3293476104736328, 0.7309755086898804, 1.0862171649932861, -0.31955623626708984, -1.0712010860443115, -1.1615777015686035, -0.9927023649215698, -0.3688812851905823, -1.0600898265838623, -1.3370747566223145, -0.6725300550460815, -0.09932851046323776, 0.8208441734313965, 1.0279016494750977, 0.45403778553009033, -0.25045040249824524, 0.3528408706188202, 2.6797027587890625, -0.5954958200454712, 1.844114899635315, 0.23634959757328033, 0.018222467973828316, 0.1441381573677063, 0.6417471766471863, 0.45456185936927795], [0.128901407122612, 1.4073234796524048, 0.3870912790298462, 1.1685518026351929, -0.04789241775870323, -0.2539207637310028, 0.0541057214140892, 1.449781060218811, -0.7618741989135742, -0.6250048875808716, -0.3060745298862457, 1.1424839496612549, -0.1922876089811325, -0.1465177834033966, 0.7697203755378723, -0.8279945254325867, -0.8079437613487244, 0.5586026906967163, 0.20610487461090088, -0.5181407928466797, -0.7247483134269714, 1.561605453491211, 0.6306738257408142, -1.3168716430664062, 0.7403689622879028, -0.03495566174387932, -1.8145736455917358, 0.25571271777153015, 1.3711185455322266, -0.5727388262748718, 0.5314807295799255, 1.484947681427002, 0.36731454730033875, 0.20901328325271606, 0.09907592833042145, 0.7110852599143982, -0.6712210774421692, 1.0854274034500122, 0.00523233599960804, 0.43717125058174133, 1.2111066579818726, 0.12642614543437958, 1.9801876544952393, 0.38125765323638916, 0.4504868984222412, -1.900686502456665, 0.6639097332954407, 0.4596642553806305, -0.8575565218925476, 0.14595927298069, 0.4631918966770172, -0.8884715437889099, 0.7609630823135376, -0.8380205631256104, -0.3463420271873474, -1.0523687601089478, 0.44889315962791443, -0.525715708732605, -2.31687068939209, -1.2154555320739746, 0.940247118473053, -1.437432050704956, 0.8236269354820251, 0.1847090721130371], [0.0017935517244040966, 1.160438060760498, 0.2227081060409546, 0.3912242352962494, 0.8781638741493225, 0.25984999537467957, 0.10507063567638397, 1.7392761707305908, 0.9504179358482361, 0.1612318903207779, 0.1744455099105835, 1.7396494150161743, 0.9770851731300354, -1.5008516311645508, 1.1424082517623901, 0.19233465194702148, -0.9184815287590027, 0.19784556329250336, 0.18076777458190918, 0.19800476729869843, -1.7343555688858032, -0.02760181576013565, 0.29817813634872437, 0.6714404225349426, -0.7068020105361938, 0.5118866562843323, -1.0530222654342651, 0.08572766929864883, 0.17326921224594116, 1.1590611934661865, 0.3808855712413788, 0.7575575709342957, -0.7579262852668762, 2.298412799835205, -0.8579951524734497, 0.004717773757874966, -0.5707851052284241, -0.047432251274585724, 1.2860935926437378, -1.979182243347168, -0.25659677386283875, -1.066030502319336, 0.057372137904167175, 0.3149799704551697, -0.27848416566848755, -0.801581084728241, 0.5808208584785461, -1.6304194927215576, -0.23694656789302826, -0.2297968715429306, 1.3117002248764038, -0.7221397757530212, 1.4032189846038818, 0.7617804408073425, 0.5979216694831848, -0.055891428142786026, -0.49942946434020996, -1.0437333583831787, -1.149691104888916, -1.338547706604004, 0.7019461393356323, -0.891723096370697, 0.22663246095180511, -0.07430609315633774], [-9.10101953195408e-05, 0.31142961978912354, 0.5286040902137756, 0.01818494312465191, 0.38240355253219604, 1.0907825231552124, 2.7341184616088867, 0.7167456150054932, 1.1222209930419922, 0.38560178875923157, 0.2948475182056427, -1.0181983709335327, 1.5674118995666504, -0.8815379738807678, -0.7833451628684998, 0.3611541986465454, 1.2958030700683594, -0.08426333963871002, 0.3273809850215912, -0.09990023821592331, 0.9633246660232544, 0.40865135192871094, 0.631662905216217, -0.742131233215332, -1.2246358394622803, 0.2259906381368637, -0.45772960782051086, -0.8351762890815735, 0.24940960109233856, -0.8391045331954956, 1.4828397035598755, -0.05587853863835335, 0.7419386506080627, 1.5326731204986572, 0.7374204397201538, -0.22291669249534607, 0.7942863702774048, 0.36340832710266113, 0.27393797039985657, -0.345041424036026, 0.7544010877609253, 0.35441628098487854, -0.6903352737426758, -0.12530416250228882, -1.3416041135787964, 0.7839745879173279, 2.3820347785949707, -1.866422176361084, -0.9960317015647888, 0.8950045704841614, -0.041090503334999084, -2.9847166538238525, -0.29798224568367004, -0.9253097176551819, 1.4600026607513428, 0.37404775619506836, 0.3059435188770294, -0.6061809062957764, -1.4303678274154663, -1.1068207025527954, 1.1415706872940063, -0.359836608171463, 0.9033340215682983, 0.28540369868278503], [-0.4060782790184021, 0.224679097533226, -1.0112786293029785, -0.07547003030776978, -0.32742881774902344, -1.2387043237686157, -0.9778695106506348, 0.5851593613624573, 0.5315726399421692, -0.0557631179690361, 0.10992532968521118, -0.7857304811477661, -0.0216514952480793, 0.06302765756845474, -0.40965062379837036, 0.7313969731330872, 1.3538252115249634, -0.9270424842834473, 0.8491361141204834, 1.136859655380249, 0.3169858157634735, -1.9366047382354736, 0.20278982818126678, -0.022071918472647667, 0.11607713997364044, -0.4254074692726135, -0.22015602886676788, -0.21524719893932343, 0.34837502241134644, 0.5061982870101929, 0.23987318575382233, 0.4314761757850647, -0.8838452100753784, 1.0387669801712036, 0.7842273116111755, -0.2449944019317627, -0.4546375870704651, -0.6653532385826111, -0.09562098234891891, -0.3966456651687622, 1.3120378255844116, 1.2846901416778564, 0.602642834186554, 0.604257345199585, -0.23912347853183746, 1.2113969326019287, -0.4712935984134674, 1.27549147605896, 1.228407621383667, 0.298016756772995, 1.2792961597442627, 1.647452712059021, -0.7260147333145142, 0.6763384938240051, 0.03767141327261925, -0.5345492362976074, 0.9167225360870361, -1.905371069908142, 0.8447445034980774, -0.7168354988098145, 1.8853933811187744, 1.0937644243240356, -0.16776026785373688, 1.0101264715194702], [-1.2660019397735596, 0.095818892121315, -0.11923360079526901, 0.21069476008415222, 0.2419121265411377, 0.09508204460144043, -0.18159276247024536, 0.5525159239768982, -0.1736522614955902, -0.7320507764816284, -1.7289094924926758, 1.5136737823486328, 1.0814400911331177, -0.3335394561290741, 1.1735814809799194, 1.869676113128662, 1.4240506887435913, 0.4182687997817993, 1.0215342044830322, 1.3587850332260132, 1.5115468502044678, -1.1379923820495605, 0.45307615399360657, -1.0802265405654907, -1.053539752960205, -0.580529510974884, 0.09512608498334885, -1.5318882465362549, -1.4190163612365723, 0.07891678810119629, 1.2365875244140625, 0.18851706385612488, -0.3390284776687622, -0.8460855484008789, -0.6588734984397888, -1.1664447784423828, 2.097470760345459, 1.2148170471191406, 0.42087870836257935, 2.49027419090271, -1.3222522735595703, 1.5942347049713135, 1.5480011701583862, -0.3385096490383148, -1.024815320968628, 0.08823171257972717, -0.38035154342651367, 0.7954931259155273, 1.2481374740600586, 0.6144508719444275, 0.30599844455718994, 1.3675802946090698, 0.3917756974697113, -1.0258344411849976, 0.23922452330589294, 1.7016180753707886, 0.42307355999946594, -0.3347458243370056, 0.49087169766426086, 1.8662322759628296, 0.5482475757598877, 0.5766382813453674, 0.9847940802574158, 0.9705456495285034], [0.8728249073028564, -0.6741747260093689, -0.06522715091705322, -0.6714591383934021, -0.4581189453601837, 0.30329564213752747, 0.9630514979362488, -0.05518738180398941, -1.359635353088379, -0.09699308127164841, 1.920689582824707, 0.008044794201850891, -0.8699445128440857, 0.49954140186309814, 0.07362309098243713, -0.13391008973121643, 0.7541250586509705, -0.8685197234153748, -1.5163768529891968, 2.672745704650879, -1.7918089628219604, 0.981592059135437, 1.5200086832046509, 0.30468541383743286, 0.29963675141334534, -0.8268015384674072, 1.2151480913162231, 1.758966326713562, -0.6457761526107788, -0.513552725315094, 0.6478553414344788, 0.963546633720398, 0.00011802239896496758, 0.18677200376987457, -0.3817693591117859, 0.4950556457042694, -0.16695867478847504, -0.15335650742053986, -1.1674927473068237, -1.1005553007125854, -1.0738192796707153, 0.037689171731472015, 1.2848719358444214, -0.5061492919921875, -0.3820897936820984, -1.410136342048645, -1.3250504732131958, 0.021462738513946533, -0.30318206548690796, 0.5992878079414368, 0.36489811539649963, 2.4777657985687256, -0.19266943633556366, 0.2039971500635147, -1.1553466320037842, 0.3365814685821533, -0.18291546404361725, -0.46783778071403503, 0.8047330379486084, 0.41638803482055664, 1.2948018312454224, 1.9291144609451294, -0.9062867760658264, -1.4823535680770874], [-0.3167349398136139, 0.8080533742904663, 0.2554488778114319, -0.5535569787025452, -0.7743794322013855, 0.47632306814193726, -0.7211197018623352, -0.8468011617660522, -0.9022165536880493, -0.4421893358230591, -0.2727770209312439, 0.5541673302650452, 0.2351258546113968, -0.9603722095489502, -1.152651071548462, -0.30428794026374817, -0.5864428281784058, 0.21853245794773102, 0.4746937155723572, 1.7330207824707031, 1.173553228378296, 0.30928120017051697, 0.5104513168334961, -0.050317686051130295, 0.2401878833770752, -0.6229442954063416, 0.09953979402780533, 0.45797282457351685, 0.42415857315063477, 0.07369454205036163, 0.8008941411972046, 0.8018917441368103, -0.15104882419109344, -0.5425220727920532, 0.23117129504680634, 0.6370586156845093, 1.8604168891906738, -0.6509881615638733, 1.4077664613723755, 0.591415524482727, 0.686383068561554, 0.7530868649482727, 0.2949313223361969, 0.41823697090148926, -0.6322723627090454, -1.0962457656860352, -0.20640917122364044, -1.054503083229065, -0.7084490060806274, 0.9694058299064636, -1.0273202657699585, 0.38540780544281006, 0.2637892961502075, -2.9309396743774414, -1.3391886949539185, 0.24612769484519958, -0.5611812472343445, -1.2759662866592407, -1.738183856010437, -0.19555923342704773, -0.008086764253675938, 0.9939209818840027, -0.917625367641449, 0.07354715466499329], [2.21461820602417, -0.6387324929237366, -0.38913917541503906, -1.3923019170761108, -1.659582257270813, 0.25218915939331055, 1.126057505607605, -0.05877590924501419, -0.08890100568532944, 0.5446717143058777, -0.19117292761802673, 0.1262529045343399, -1.1809769868850708, -0.19935616850852966, -1.4179545640945435, -0.41703659296035767, -0.9468274712562561, 0.17250196635723114, 0.9003942012786865, 0.4889276325702667, -0.07649869471788406, -0.10059506446123123, 1.6287873983383179, 0.12962418794631958, -0.7972092032432556, -0.8367069363594055, -0.767153263092041, 0.3504214882850647, -3.088937997817993, 0.11224659532308578, -1.5645396709442139, -0.7496448755264282, -0.09349918365478516, -0.27755147218704224, 0.00026760855689644814, 0.7186422944068909, -1.9759547710418701, 0.23413577675819397, -0.06338127702474594, -0.38439252972602844, 0.02145608328282833, 1.014839768409729, 2.1070609092712402, 0.9241929650306702, -0.21023815870285034, -0.03147092089056969, -0.04491990804672241, 2.0806772708892822, 0.8388354182243347, -0.11959230154752731, 0.9302775859832764, -0.30544590950012207, 1.8465156555175781, -0.805335283279419, 1.8822687864303589, -0.4935730993747711, 0.05246153101325035, -1.3166439533233643, -0.5479817986488342, 1.2904305458068848, -0.4898298978805542, -0.5339193940162659, -0.26172196865081787, -0.8456873297691345], [-1.1956768035888672, 0.16546641290187836, -0.6276174187660217, 0.5276662111282349, -0.43474704027175903, 1.1965521574020386, 0.43426811695098877, 1.6859424114227295, 0.7514024376869202, 0.33584779500961304, -0.8475509881973267, 0.02057052031159401, 0.1569412350654602, 1.4566560983657837, 1.0923222303390503, 0.7129734754562378, 1.9924449920654297, -1.8419396877288818, 0.6114763617515564, -0.6998757123947144, 2.089704990386963, -1.2194950580596924, 0.1359119713306427, 0.44376590847969055, 1.2583345174789429, 0.043976109474897385, -1.351686954498291, -1.0163391828536987, 0.1374116837978363, 0.7041248679161072, -1.4904465675354004, 0.6364693641662598, 0.9300070405006409, -0.14326117932796478, 0.49634742736816406, 0.9013074040412903, 0.11471480876207352, 0.034721456468105316, 0.913844883441925, 0.6406576037406921, -0.4121788740158081, 0.6841332912445068, 0.7914264798164368, 0.14603526890277863, -0.9439556002616882, 0.6273266673088074, -1.033179759979248, 0.30545833706855774, 0.263578861951828, -0.19290746748447418, -0.6465483903884888, -0.4328314960002899, 1.3989319801330566, 0.6090878248214722, -0.4985743761062622, -0.7046378254890442, 0.8588829636573792, -0.3419557809829712, -1.6054593324661255, -1.4719656705856323, 0.4632362723350525, -1.4406776428222656, -0.36166220903396606, -0.7316349744796753], [0.7358916997909546, 0.91306471824646, -0.1888262927532196, -1.4504899978637695, -0.5454221963882446, 0.8899974822998047, -0.2919712960720062, 0.9863013029098511, -1.5344717502593994, 1.5105408430099487, -1.5626713037490845, 0.9652763605117798, -0.5336992144584656, -0.5548791885375977, 0.07351045310497284, 1.2010395526885986, -0.15192805230617523, -0.4832584261894226, 2.2782094478607178, 0.6634730696678162, -0.9619861245155334, -0.8564246296882629, 0.16454680263996124, 0.39724525809288025, -1.7212952375411987, -0.7598056197166443, -0.7395023107528687, 0.9422680139541626, 0.009832441806793213, 1.5342365503311157, -0.46789684891700745, -2.3765454292297363, 0.3157302141189575, -0.2127630114555359, -1.05728280544281, -0.872896134853363, 0.10360566526651382, 0.23559963703155518, -1.5060009956359863, 0.8502399921417236, -0.12406259030103683, 0.20048010349273682, -2.003743886947632, 0.5103932023048401, -0.6578142046928406, 0.6198625564575195, 1.6946321725845337, 2.2460591793060303, -0.2795705795288086, -2.3515443801879883, 1.1350218057632446, 1.7566500902175903, -0.5171242356300354, -1.4312528371810913, 0.19762517511844635, -0.11621512472629547, -0.8430367112159729, 1.5394011735916138, 0.2065090388059616, 1.078161358833313, 0.23547819256782532, -1.0277522802352905, -1.277742862701416, -0.9149284362792969], [2.212785005569458, 0.6327153444290161, -0.041987042874097824, -0.17325814068317413, 1.2612547874450684, -0.43275654315948486, 0.4271729588508606, 0.3494953513145447, -0.43542081117630005, 0.2546551525592804, 1.034006118774414, 1.2592352628707886, -0.3528693616390228, -0.8265895247459412, 0.2864569127559662, 0.20287124812602997, -0.37717923521995544, -0.14293740689754486, -0.7833130955696106, 0.13581973314285278, -0.46478044986724854, 0.695793867111206, 0.5816047191619873, -1.5175482034683228, 0.19472604990005493, -0.4099003076553345, -0.9366158246994019, 0.5239181518554688, 0.08379963040351868, 1.2572954893112183, -0.5555177927017212, 0.25339019298553467, 1.4053916931152344, 0.7730682492256165, 0.5934435129165649, 0.31486549973487854, 0.8844709992408752, -0.244394913315773, -0.7098204493522644, 0.16716982424259186, -3.2584261894226074, -0.5277463793754578, -0.5738891959190369, -1.7643901109695435, -2.2648653984069824, 0.6353516578674316, -0.4504030644893646, 0.018278028815984726, 0.2052941769361496, 0.32864081859588623, -0.4441337585449219, -0.9422603249549866, -0.7340000867843628, -1.3146021366119385, -1.6420999765396118, 0.7041512727737427, 2.043890953063965, -0.45230868458747864, 0.35919561982154846, -0.7432669401168823, -0.03540346026420593, -0.14671191573143005, -0.7348085045814514, -0.11121996492147446], [0.4923586845397949, -0.028300944715738297, 0.8518080711364746, 0.6697766184806824, -1.8495018482208252, 1.6672712564468384, 0.9444161057472229, 0.6624342799186707, 0.2063034474849701, -0.42781928181648254, 0.20858410000801086, -1.1217763423919678, 0.26924023032188416, 0.051455408334732056, -0.9011332988739014, 0.2590402662754059, 0.480436235666275, -1.016469955444336, 0.8720410466194153, -1.7888931035995483, -0.04476574435830116, -0.044387318193912506, -0.012592662125825882, 2.799229383468628, 0.8984850645065308, -0.3252837359905243, -0.09666486829519272, -0.2770061194896698, -1.3765878677368164, -1.6193487644195557, 0.8105029463768005, 1.122527003288269, -0.49207964539527893, 0.33992621302604675, -0.7241904139518738, -0.18970166146755219, -0.29519858956336975, -0.23646456003189087, -1.3367537260055542, -0.15995773673057556, -0.685924768447876, -0.31366464495658875, 0.1534912884235382, -0.21107473969459534, 2.079118490219116, -0.9777989387512207, 0.478569895029068, -0.790108323097229, 0.4643545150756836, 0.5033639073371887, -0.07250330597162247, 0.6104884743690491, -0.17196284234523773, 1.6735213994979858, -0.8169301152229309, 0.3769838213920593, -0.29606202244758606, 0.39231568574905396, 1.0009819269180298, -0.7588160037994385, 0.8857430219650269, -0.6558279991149902, -0.9725530743598938, -2.0829856395721436], [-0.4083751440048218, -0.006978982128202915, -1.0866076946258545, -0.9630460143089294, -0.6751781105995178, -2.6300528049468994, 1.5245561599731445, 0.019914165139198303, 0.6292248368263245, 0.5602970719337463, -1.3043584823608398, -0.40383753180503845, -0.01979980804026127, -0.6353054642677307, 0.08972474932670593, 0.8916540145874023, 1.7713096141815186, 0.477937787771225, 0.30450132489204407, -1.3441678285598755, 0.8316695094108582, 0.6863425374031067, 0.31224384903907776, -0.06591511517763138, -0.30333346128463745, 0.8627716302871704, 0.0248735211789608, 0.4171529710292816, 1.0690431594848633, -0.2485860288143158, 0.5072341561317444, 1.363339900970459, 2.0639734268188477, 1.8811578750610352, -1.7822635173797607, -1.5431315898895264, -0.13158346712589264, 0.7855671644210815, 0.580041229724884, 1.1798263788223267, -1.033219337463379, 0.6032026410102844, 0.09698404371738434, -1.2245142459869385, -0.24270544946193695, 1.83056640625, -0.5095301866531372, 0.9821353554725647, -0.16878364980220795, -0.2331111878156662, 2.151000738143921, 1.1996879577636719, -0.941044807434082, 1.4872610569000244, -0.41383683681488037, -2.2645063400268555, 0.18725267052650452, 1.3001978397369385, 0.5881398320198059, 1.949747920036316, -0.6777576208114624, -0.1015077456831932, 0.394582599401474, 0.2891712188720703], [0.1338500678539276, -0.21137435734272003, -1.1379252672195435, 0.36560726165771484, 0.5843451619148254, -0.6951500177383423, -0.397095650434494, -1.8570771217346191, -1.9010976552963257, -1.8518346548080444, -0.8898884654045105, -0.7500206828117371, -1.8646166324615479, -0.5743189454078674, -0.6999656558036804, -1.4426920413970947, -0.9219735860824585, 0.0022654675412923098, -1.5413689613342285, -0.7983105182647705, 0.37061837315559387, 1.464141607284546, -0.01514384150505066, 1.4415346384048462, 0.0796087384223938, -2.027381420135498, -1.0148744583129883, -0.5296804904937744, -0.2725086212158203, 1.232822299003601, -0.7539992928504944, -1.277705430984497, 1.7112905979156494, -1.1405630111694336, -0.8980674147605896, -0.2664238214492798, 1.664280891418457, 0.7072256803512573, -0.4527451694011688, -0.3148724436759949, 0.6854496002197266, -1.2674931287765503, -1.0963618755340576, -0.8584434986114502, -1.0880428552627563, -1.1158738136291504, 0.678545355796814, 0.6476478576660156, -2.054962396621704, 1.101661205291748, -0.6323416233062744, -0.17097152769565582, 1.3135066032409668, -1.2565184831619263, 1.2932384014129639, 0.28170475363731384, -1.6518267393112183, -0.004501108080148697, -2.2196903228759766, 0.06671744585037231, 1.85814368724823, -0.34854111075401306, -1.3882249593734741, 0.8635414242744446], [0.4176501929759979, 1.0544618368148804, -0.13083568215370178, -2.8554487228393555, -1.9967867136001587, 0.2566010653972626, 0.4845184087753296, 1.1127334833145142, 0.4298349618911743, 0.13248798251152039, -1.3355315923690796, -0.4323112964630127, 0.8890862464904785, -1.0426732301712036, -1.3329048156738281, 1.6533973217010498, -1.7974570989608765, 1.8481254577636719, 0.010607318952679634, 0.19320133328437805, 0.5478234887123108, 0.9869384169578552, 0.6902067065238953, -0.0014418723294511437, 2.9917056560516357, -0.8979178071022034, -0.47388413548469543, -0.10583975166082382, 1.9554030895233154, -0.0756848007440567, -0.498854398727417, -2.25758957862854, -0.3994917571544647, -0.44955548644065857, 0.6470907926559448, 0.08827602863311768, 0.4251495599746704, 1.4235919713974, 0.7354801297187805, 0.6778481006622314, 0.6668644547462463, 0.16324003040790558, -1.468824028968811, 1.3434854745864868, -1.2010133266448975, -1.2481564283370972, 0.7641389966011047, 0.24249336123466492, 0.15646077692508698, -1.4123892784118652, 0.43239685893058777, 0.7920225262641907, 0.6382547616958618, -0.6279038786888123, 1.0834254026412964, -1.5568310022354126, 0.8342620134353638, -0.2642163038253784, 0.47001776099205017, -0.901355504989624, -1.1702311038970947, 0.5009081363677979, 0.33256304264068604, -2.081660270690918], [1.180738091468811, -0.4338509440422058, 2.75056529045105, -1.5949437618255615, 1.2196083068847656, 0.6550096869468689, 0.1732117384672165, 0.6011385917663574, 0.49920928478240967, -0.2686796486377716, -0.9529901742935181, -1.9077458381652832, 0.06363996118307114, 0.09208331257104874, -0.757102370262146, -0.8860362768173218, -0.1636447161436081, 1.1372102499008179, -0.045801881700754166, 1.1500356197357178, -1.7137086391448975, 1.1917039155960083, 0.5839354991912842, -1.1508948802947998, -0.9249263405799866, -0.035305798053741455, 0.9469793438911438, -0.01646721549332142, 0.8681336045265198, -0.18689052760601044, -1.712454915046692, 1.221982479095459, 0.9017845988273621, -0.44549882411956787, 0.20803824067115784, -1.1724796295166016, 1.367691993713379, 0.6488863229751587, -1.4870884418487549, 0.5881365537643433, 0.09998130053281784, 0.953865647315979, -0.35203102231025696, 0.21782691776752472, -0.006827919278293848, -0.03540503606200218, 0.4530743658542633, 0.34636586904525757, -0.1572936326265335, 0.4282582104206085, 1.1803364753723145, -1.1080650091171265, 1.2000586986541748, -0.44470056891441345, 0.40604379773139954, -0.4674798548221588, 0.5847489833831787, -0.7464117407798767, 2.1372220516204834, -0.11881285160779953, 0.3598453402519226, -1.4458935260772705, -1.5081058740615845, 0.3351280987262726], [0.5172230005264282, 1.125732660293579, 0.3855307102203369, -1.3376611471176147, -0.3725225031375885, 0.9433608651161194, -1.5986677408218384, 0.8575088977813721, 1.1802432537078857, 0.5118990540504456, -1.0337834358215332, 0.15560749173164368, 1.138393521308899, -0.5500892996788025, -0.29204487800598145, 0.8487135767936707, 0.9271007776260376, 0.6657681465148926, 0.3660956621170044, -0.6893815398216248, -0.38367006182670593, 0.3484610319137573, -1.0959302186965942, 0.38841649889945984, 0.4159221947193146, -0.5611485838890076, 0.5362919569015503, -0.7805727124214172, 0.570207417011261, 1.66506826877594, -1.8908973932266235, -0.6028265953063965, -0.7313169240951538, 0.20526212453842163, -1.5304185152053833, 0.3122512400150299, 0.15095287561416626, 0.437416672706604, -1.0975600481033325, 0.5566399097442627, -0.26235127449035645, 0.2585127055644989, -0.5492003560066223, -0.5480552911758423, 0.3518919050693512, -1.5457618236541748, -0.7483808398246765, 0.010275948792696, 0.6656513214111328, -0.8949036598205566, 0.29206937551498413, -0.6544546484947205, 0.8164030909538269, -0.2569747865200043, -1.2660353183746338, 0.1207030788064003, 0.013606484979391098, 0.8620548844337463, -0.30565398931503296, 0.28102385997772217, -0.5067719221115112, 1.3406888246536255, -1.8225044012069702, 0.6451958417892456], [0.45978546142578125, -0.995323121547699, 0.3152414560317993, 0.8816158175468445, 0.2847682237625122, -0.8193361163139343, -0.31328892707824707, -0.7404950261116028, 0.05884113535284996, -0.3235747516155243, -0.7511046528816223, -0.7979426980018616, -0.1915416568517685, -1.2191463708877563, -0.746637225151062, -0.5189638137817383, -1.1029126644134521, -0.5370946526527405, -1.178939938545227, -0.6209914088249207, -0.0716165229678154, -1.3173511028289795, -0.839043378829956, -0.44773218035697937, 0.784292995929718, 1.6648913621902466, -1.3362057209014893, 1.1549279689788818, 0.9400604963302612, -0.37829744815826416, -1.1619054079055786, -1.1065593957901, -0.016293011605739594, -0.6498944759368896, -1.2209205627441406, -1.9724030494689941, 0.323839396238327, -1.0326621532440186, 0.6327632069587708, 1.6146697998046875, -0.5153446197509766, -0.08784189820289612, -0.32850292325019836, 0.771103024482727, 0.8491606712341309, -0.0275412630289793, 0.787708044052124, 0.9953592419624329, 0.9147438406944275, 0.8477188348770142, 0.9997552633285522, 0.44065025448799133, 0.6977439522743225, 0.8965549468994141, -1.0949257612228394, -2.223670482635498, 1.1458642482757568, -2.0332164764404297, 1.2118877172470093, -1.332092523574829, 1.6707634925842285, 0.03593122959136963, -0.3673110008239746, 0.15511327981948853], [-1.2498321533203125, 0.9190027713775635, 1.5109853744506836, 0.368425190448761, 1.610855221748352, 1.4206016063690186, 2.443777322769165, 1.653482437133789, -1.3885551691055298, -0.2113371342420578, -0.076836958527565, -1.0651427507400513, -0.6563759446144104, 1.0955665111541748, 0.2697369158267975, -1.0982873439788818, -0.4673323333263397, 0.4516561031341553, 1.0270025730133057, -0.5627550482749939, 0.8179206252098083, -0.8398066759109497, -1.1278728246688843, -0.5962072610855103, -0.7049278616905212, -0.4541032314300537, -0.12549638748168945, 0.2242361605167389, -1.1281589269638062, -0.24730104207992554, 0.43192312121391296, 0.07238121330738068, 0.6517958641052246, -0.4345276355743408, -0.03264299035072327, 1.013791799545288, -0.7568280100822449, -0.46484366059303284, 1.6432230472564697, -0.6183319091796875, 0.06469637900590897, -0.9696466326713562, 0.08610130846500397, 1.5233755111694336, 0.5168229341506958, -0.5630276799201965, -0.32997554540634155, 0.22822195291519165, -0.5156564116477966, 0.7464662790298462, -1.5683677196502686, -1.5478873252868652, 0.4823416769504547, 0.5056772232055664, 0.29230448603630066, 1.364579677581787, -2.4314534664154053, 0.3930527865886688, -0.004492033738642931, -0.348061203956604, 0.9106738567352295, 1.055991530418396, 0.6614327430725098, 0.932145357131958], [-0.039971813559532166, -1.658424973487854, 2.113250970840454, 1.4198832511901855, 0.5830320119857788, 1.4931244850158691, -0.9193474054336548, -0.004374528303742409, -0.6007652282714844, -1.1834421157836914, -0.5897128582000732, -0.6370768547058105, -0.6828294396400452, 0.8693879842758179, -1.3646758794784546, 0.0055992985144257545, -1.8513202667236328, -1.3034340143203735, -0.42159929871559143, 0.4504390358924866, -0.48504966497421265, -1.0208977460861206, 0.8700759410858154, -0.2917236387729645, 0.13990308344364166, 0.3978074789047241, 0.1501636505126953, 0.13308365643024445, -1.114830493927002, 0.85426926612854, 3.544802665710449, 2.1723389625549316, -1.9371660947799683, 0.8594525456428528, 1.8439017534255981, 1.632893681526184, 0.23608051240444183, -1.154934287071228, -0.9023321866989136, 0.33226993680000305, -0.796623170375824, 1.5439194440841675, 1.3745672702789307, 0.3226720690727234, -0.06849893927574158, -0.30573129653930664, -0.746601402759552, 0.3910447657108307, 0.08354402333498001, -0.5562431812286377, -1.6431375741958618, 1.1051002740859985, 0.7033380270004272, 0.717616081237793, -0.6172041893005371, 0.6191536784172058, -1.72321617603302, -0.5701397657394409, 0.9463244676589966, -0.7081565260887146, -0.21126846969127655, -0.04854293912649155, -0.3085874021053314, -0.4049314260482788], [0.689306914806366, 0.08021196722984314, 1.7726995944976807, -0.7156329154968262, 0.12081358581781387, -1.8879895210266113, -0.3752502501010895, 1.6392273902893066, 0.17791837453842163, -0.4479845464229584, -0.24483171105384827, 0.5706183314323425, -2.1448514461517334, 1.516535997390747, -0.09092666208744049, -0.8105781674385071, 1.6963374614715576, -1.749741554260254, 1.7736796140670776, 1.149064064025879, -0.7942481637001038, -1.9084750413894653, -0.41913336515426636, 1.3167895078659058, -1.8982422351837158, -0.2898845672607422, 0.3983781337738037, -0.31165552139282227, -0.5521453619003296, 0.19261212646961212, -1.2959626913070679, -0.4462892413139343, 0.9141874313354492, 1.951599359512329, 0.6813660264015198, 0.9072195887565613, -1.7760136127471924, -0.31973350048065186, 0.18386177718639374, -0.98513263463974, -0.42568591237068176, 0.6738311648368835, -0.6444446444511414, 1.211530089378357, 1.4181623458862305, -1.2324498891830444, -0.5630785822868347, -1.2880299091339111, -0.3409503698348999, -0.13597339391708374, 1.3197121620178223, -0.5722538828849792, -1.3192379474639893, -0.835088312625885, 1.1618037223815918, -0.16415828466415405, 0.061142370104789734, 1.1971584558486938, -1.0706623792648315, 1.1131465435028076, 1.528045415878296, -0.24749961495399475, 0.5488210320472717, 0.5333590507507324], [-0.28061601519584656, 0.7781273722648621, -0.1435827910900116, -1.7780632972717285, 0.6878049969673157, 0.4390887916088104, -0.5905547142028809, -0.1451546996831894, -0.2521609961986542, 0.9479840397834778, 0.6199293732643127, 0.5807692408561707, 0.2917172610759735, 0.43595096468925476, -1.627489686012268, -0.2967783808708191, -0.9653513431549072, 0.8793544173240662, -1.476235270500183, 2.7938740253448486, -1.3135579824447632, 0.6694203615188599, 0.3343348503112793, -0.006717171519994736, -1.4052326679229736, 2.1670196056365967, -0.0760750025510788, 0.7012491226196289, -0.9453780055046082, 0.19766592979431152, 2.8411049842834473, 0.4573350250720978, -0.982905387878418, -0.6988832950592041, 0.4431167542934418, -0.34675049781799316, -0.7310147285461426, -0.23716393113136292, 0.5817084908485413, -1.194018006324768, 0.19734857976436615, 0.878284752368927, -1.9374605417251587, 1.0841096639633179, 0.3179456293582916, -0.4559800624847412, 1.8865373134613037, 1.663336157798767, 1.1799321174621582, 1.233201503753662, 0.47410696744918823, -0.22064876556396484, -0.798986554145813, -1.6201311349868774, 0.3537547290325165, -0.9571784734725952, -1.9600785970687866, -0.06181269884109497, 0.6878042817115784, -0.19269342720508575, -0.350477010011673, -0.8606385588645935, 1.1021603345870972, -1.4652493000030518], [-0.7023875117301941, 0.5730331540107727, 0.2696608006954193, -1.4723330736160278, 0.19744516909122467, 0.14143764972686768, 1.8010183572769165, -1.791796088218689, -0.8586370944976807, -1.2169102430343628, -1.5934576988220215, -0.7008548974990845, -0.547340989112854, -2.79856538772583, -0.6231555938720703, 1.5390352010726929, -0.053044047206640244, 0.21757234632968903, 1.1469134092330933, 0.011586223728954792, -1.1423333883285522, 0.5305763483047485, -2.1070945262908936, 0.13407345116138458, -0.699680507183075, -0.6194537281990051, -0.24904674291610718, -1.072004795074463, -1.1244083642959595, -1.2682496309280396, -0.3703034520149231, 2.5132863521575928, -0.6877566576004028, -0.18705952167510986, -0.2312370389699936, 0.33461350202560425, -0.2981465756893158, 0.23340094089508057, 0.8888617753982544, 0.914564847946167, 0.4787363111972809, -0.6894596815109253, -0.5191622972488403, -0.615074872970581, 0.0852278620004654, -0.6858075261116028, 1.5149823427200317, -0.22460773587226868, 0.7577795386314392, 1.955333948135376, 2.260935068130493, -0.7894863486289978, -1.0135223865509033, -1.9642866849899292, 0.2490386962890625, -0.29950132966041565, -1.470112919807434, 0.7450180053710938, -0.8271400928497314, -1.420181155204773, 0.1701509803533554, -1.7929511070251465, 0.16419284045696259, 0.30614492297172546], [-0.8818769454956055, 1.5947284698486328, -0.7128564715385437, -0.622226893901825, 0.3905712366104126, -1.2059381008148193, 0.78249591588974, -0.7500389218330383, -1.0010348558425903, -1.510216236114502, -0.11220266669988632, -0.8251012563705444, -0.4518013000488281, 0.017281951382756233, 1.5457587242126465, -0.4180561304092407, 0.5355880260467529, 0.3770081400871277, -1.1452150344848633, -0.2069728672504425, 0.5851542949676514, -0.25272732973098755, 1.0775550603866577, 0.22693347930908203, 1.0660327672958374, -0.05106176435947418, 0.3654868006706238, 0.44011348485946655, -0.16497622430324554, 0.06886817514896393, 0.8387616276741028, 0.430001437664032, -1.7612905502319336, -1.0638186931610107, 1.347975254058838, -0.5620030760765076, 0.25674235820770264, 0.3242359459400177, 1.5869594812393188, 0.008677260018885136, -0.17728812992572784, -1.3673534393310547, -1.1221020221710205, 0.8597489595413208, -0.5903713703155518, 0.7704781293869019, 0.501920759677887, 1.398230791091919, -0.2792001962661743, -0.2140377312898636, -0.7439671158790588, -1.133731722831726, -0.5743463039398193, 1.7421246767044067, 1.2136142253875732, -1.9559794664382935, 0.773404061794281, 2.2128658294677734, 0.3288072347640991, -0.3858877122402191, 0.046995390206575394, 0.3578617572784424, 1.781737208366394, 0.879611611366272], [-1.7900786399841309, -0.9195627570152283, 0.9220048785209656, -1.4292330741882324, 0.9055185914039612, 0.548747181892395, -0.1120673343539238, -0.033178284764289856, 0.7911694049835205, 0.3214140236377716, -0.31179314851760864, -0.024238504469394684, 1.4834319353103638, 0.1543201357126236, 2.0398006439208984, -1.6411323547363281, 1.0439273118972778, 0.14559681713581085, 0.4316193461418152, -0.5948046445846558, -1.271727442741394, -1.6566051244735718, -0.0236977506428957, 0.8041830658912659, 0.6252023577690125, 0.33414843678474426, 0.19529825448989868, 0.8004851937294006, 0.04345807433128357, 1.4509836435317993, -0.8411967754364014, 0.967863142490387, -0.46542540192604065, -0.8336730003356934, 0.11965376883745193, 0.6519154906272888, 0.20026198029518127, -1.265955924987793, 0.8884543180465698, 0.8614450693130493, -0.1830921471118927, -0.28154680132865906, 0.07774189114570618, -1.3351296186447144, -0.18943801522254944, 1.2353718280792236, 0.8850526213645935, 0.2442958503961563, -1.1147408485412598, -0.7238436937332153, -0.5447007417678833, -0.7400457859039307, -0.013178753666579723, -1.2621020078659058, -1.5817197561264038, 0.5456722974777222, 0.7473388314247131, -0.1505802869796753, 1.4925752878189087, 0.8837418556213379, -0.268492192029953, 0.07705076038837433, 0.9084384441375732, 0.21305666863918304], [0.2697577476501465, 2.2449421882629395, -0.5915483236312866, 0.02598041109740734, 1.8663190603256226, 0.22112570703029633, 0.3991040289402008, -1.0826759338378906, 1.6022844314575195, 0.31975165009498596, 1.330852746963501, 0.7726818919181824, -1.4903820753097534, 2.3876359462738037, 1.2242194414138794, 0.17373165488243103, -0.9577491879463196, -0.9486648440361023, -0.25776615738868713, -0.20094314217567444, 0.5028842091560364, 0.2030819207429886, -0.05279321223497391, 0.6204717755317688, -0.15430565178394318, 0.21685539186000824, -0.01986885629594326, -0.3358728587627411, -0.199166938662529, 0.7226229310035706, 0.5617087483406067, 0.5422030091285706, -1.3721137046813965, -2.375317096710205, 0.9002093076705933, 0.6995083689689636, -0.24659039080142975, 1.8560056686401367, -0.22437359392642975, -0.5258949398994446, 0.5594707727432251, -0.8856609463691711, 3.084646463394165, -1.0640757083892822, -0.8238375186920166, -0.919736385345459, 0.8635029792785645, -0.47783029079437256, -0.5092834830284119, -0.1574515700340271, -0.4317828416824341, 0.14955173432826996, -0.6784935593605042, -1.4703985452651978, 0.30740687251091003, -1.8655081987380981, 1.0106806755065918, 0.31959110498428345, 1.1654012203216553, 1.3635834455490112, -0.9415153861045837, -0.5966653823852539, 1.1601096391677856, -1.7819430828094482], [-0.16004320979118347, -0.11814378201961517, 0.3382108509540558, -1.3611711263656616, -0.7027940154075623, -1.5273237228393555, -0.8559970855712891, -0.06422016769647598, 0.07045672088861465, -0.14350353181362152, -0.6138055324554443, -0.8771029114723206, 0.047209251672029495, 0.5591081380844116, -1.1025960445404053, 1.8518105745315552, 0.11820748448371887, -0.58627849817276, -0.6306551098823547, 0.0193829033523798, 0.5478807687759399, 0.5614979267120361, 0.029314439743757248, -0.8598300218582153, 1.2017688751220703, 0.017083818092942238, -0.29777035117149353, 1.3861786127090454, 1.837063193321228, -0.4619734585285187, 0.19089113175868988, 0.9414905905723572, -1.299164891242981, 1.0919846296310425, 0.7496779561042786, -0.010179422795772552, 0.6495978236198425, -0.09066270291805267, -2.159205198287964, -0.9268139600753784, 0.6956284642219543, -0.012958302162587643, -1.2371680736541748, -0.35600289702415466, 0.7344687581062317, -0.775566041469574, 1.0190455913543701, 0.7411176562309265, 0.3410598337650299, 0.813362717628479, -0.9658582806587219, 0.17524228990077972, -0.5927488207817078, -1.5006242990493774, 0.44495484232902527, 1.2448594570159912, -0.9267497062683105, -1.2313987016677856, 0.7491569519042969, 1.086682915687561, -0.4252524971961975, 1.4506181478500366, 1.5184916257858276, -1.9673497676849365], [-0.9971127510070801, 1.326066493988037, -0.5971192121505737, -0.3467038869857788, -0.020257167518138885, -0.2748483121395111, -1.1722760200500488, 1.0485979318618774, -0.11878462135791779, 0.44391918182373047, 1.5311646461486816, -0.37757083773612976, 0.42491868138313293, -0.5852200388908386, 0.14710624516010284, 0.2837800979614258, -2.150893211364746, 0.6854469180107117, 0.9505962133407593, 0.7969160676002502, 0.281024694442749, -1.188783884048462, 0.18383167684078217, 0.5382936596870422, -0.26172658801078796, -0.8699862957000732, 0.03097275272011757, -0.8113969564437866, -0.6061572432518005, 1.6818501949310303, -2.2671287059783936, -0.5701040625572205, -0.49323412775993347, -2.0235273838043213, -0.007568285800516605, 1.3075040578842163, 0.8150373697280884, -0.378537118434906, -1.165642499923706, -0.8946046829223633, 0.915195643901825, 1.0388678312301636, 0.048943616449832916, 0.8515879511833191, -1.1329948902130127, -0.10273633897304535, 0.9538130760192871, 0.35015183687210083, -0.3004085123538971, 1.2473818063735962, 1.270729660987854, -2.284214973449707, -0.46363765001296997, 1.4420431852340698, 0.7537528276443481, 1.368523120880127, 0.11144296824932098, 0.3988091051578522, -0.6392759084701538, 0.13295066356658936, 0.6148043870925903, -0.8959694504737854, -0.9744212627410889, -1.2070717811584473], [0.455527126789093, 0.6147217154502869, -1.0294251441955566, 1.160995602607727, 0.06821942329406738, -1.4277712106704712, 0.4892619550228119, 1.2067527770996094, -0.35013264417648315, 1.1460843086242676, 0.548748254776001, 1.5961905717849731, 1.154118299484253, 0.5095701217651367, -0.07184401899576187, 1.0822116136550903, -0.4653591811656952, -0.5298620462417603, -1.1922736167907715, 0.8825876712799072, -1.7694274187088013, 1.1722743511199951, -2.080310106277466, -0.9759420156478882, -1.117720365524292, 0.14937803149223328, -1.4429608583450317, -0.01563430391252041, -1.2850878238677979, 0.3093147873878479, 0.6343069076538086, -1.1464887857437134, -0.47607457637786865, 0.34084784984588623, -0.40216919779777527, 0.5199253559112549, -1.4774154424667358, 1.2946830987930298, -0.5080097913742065, -0.6568661332130432, -1.2212897539138794, 0.25711163878440857, -0.08846446871757507, 0.5742314457893372, 0.2882190942764282, 0.2839752435684204, -1.4326740503311157, -0.8894423246383667, 0.7412011623382568, 0.6975820660591125, -0.09359437227249146, -0.8315865993499756, -1.2695857286453247, -1.4815707206726074, -0.06540385633707047, 1.3441658020019531, 0.06789802759885788, -0.31638792157173157, -0.10123567283153534, -0.46090173721313477, 0.6497552990913391, 0.7801120281219482, -0.5364017486572266, 2.8599629402160645], [0.10508638620376587, 0.06745877116918564, -0.7485043406486511, 1.1692768335342407, 0.3849138915538788, -0.6088376045227051, -0.2048639953136444, 1.0699336528778076, -1.908221960067749, 1.2572687864303589, 0.43551942706108093, -0.8499947190284729, -1.8607866764068604, 0.7138776183128357, -1.403521180152893, 0.015181479044258595, -0.2223062664270401, 0.7893797159194946, 0.2464161515235901, 0.032408908009529114, 0.48668015003204346, 1.5306212902069092, -0.26611968874931335, 0.4500775635242462, -0.06254824995994568, -1.5751510858535767, 0.5321662425994873, -1.0480635166168213, -1.268546462059021, -1.1159220933914185, 1.6364710330963135, 1.0263946056365967, -0.398074209690094, -1.0263440608978271, -1.4965310096740723, -0.14756855368614197, -0.8910863399505615, 0.4003511071205139, 0.9794723391532898, 1.263153076171875, -1.211475133895874, 0.8394893407821655, -0.9074456691741943, -0.8626623749732971, -0.27675673365592957, -0.43337076902389526, 0.7368846535682678, 0.526846706867218, 1.5722428560256958, 0.14907531440258026, 0.992428183555603, -0.6590901017189026, 1.640811800956726, -0.8919944167137146, -0.7320170998573303, -1.1066334247589111, 1.6289739608764648, -0.7635530829429626, -0.11774443089962006, -0.7404131889343262, 0.02106480672955513, -1.2202911376953125, 0.10173138976097107, 0.30678480863571167], [1.7002826929092407, 0.8932514190673828, -0.7537760138511658, -0.9719266891479492, -1.3198031187057495, -0.07060755044221878, -0.7173315286636353, -1.395788311958313, -1.8513597249984741, 0.6178365349769592, 1.4275318384170532, 1.177642583847046, -1.4988853931427002, -0.4959982931613922, 0.3576614558696747, -0.5604247450828552, -1.526251196861267, -1.2953031063079834, 1.1204922199249268, -0.9708876013755798, -0.02513168193399906, -0.48209482431411743, -0.9166572093963623, 0.04417163133621216, 0.712311327457428, -0.9720004200935364, -0.24489495158195496, -0.3882567286491394, 1.0193464756011963, 0.029880570247769356, -1.566582202911377, -0.8651622533798218, -0.4309893548488617, 1.4801067113876343, -0.9924131035804749, 0.5968056917190552, 0.05925549566745758, -0.8656046390533447, 0.4615054726600647, 1.3523973226547241, -0.37255439162254333, 0.3611217141151428, -0.9854997992515564, -0.24130208790302277, 1.3868408203125, 0.3291398882865906, 0.11517776548862457, 1.381593942642212, -0.7688442468643188, 0.19245916604995728, 0.6129166483879089, 1.1840755939483643, -0.32262861728668213, -0.12434465438127518, 1.6428121328353882, -1.0258903503417969, -0.3171800673007965, 0.1126973107457161, 0.18237677216529846, -0.7995702624320984, 1.105989694595337, 0.00998762622475624, 0.18041114509105682, 1.4637664556503296], [1.4035508632659912, 0.8227208256721497, 0.9040852785110474, 2.9576404094696045, 1.1075624227523804, -0.4813080430030823, -0.7194573879241943, -0.8919242024421692, -2.0495893955230713, 0.8294411897659302, -0.6389020085334778, 1.8633670806884766, -0.18491733074188232, 0.585809051990509, 2.363579750061035, 0.7341932058334351, -0.8007022142410278, -1.5915451049804688, -0.32698193192481995, -0.4486667215824127, 0.32237520813941956, -0.0064767757430672646, -1.9410218000411987, -0.44487321376800537, -0.8303346633911133, -0.5899572372436523, -0.821876585483551, -0.8833673000335693, 0.032731425017118454, -0.957970142364502, -1.1936124563217163, 0.6881375908851624, -0.6565336585044861, -0.45305880904197693, 1.311975121498108, 0.5328494310379028, 1.3246959447860718, -0.829612135887146, -1.0862282514572144, -0.5752453804016113, 0.17953336238861084, -0.4955666959285736, -1.8761980533599854, -0.41369789838790894, 0.2183685302734375, -0.7954877614974976, 1.090126872062683, 0.7592087984085083, 0.06866152584552765, -0.7560296058654785, -1.3662943840026855, -0.17194095253944397, -1.1645303964614868, -0.5806599855422974, 1.4508349895477295, -1.4111042022705078, 0.6313235759735107, 1.348578929901123, -1.606117844581604, 0.6959260702133179, 0.9251189827919006, -0.9227353930473328, 1.342740535736084, 0.5661807656288147], [-0.49476829171180725, 1.7365577220916748, 0.26136791706085205, 1.1202282905578613, -0.2282055765390396, 0.5677645206451416, 1.0276449918746948, 0.42515426874160767, -0.0030480637215077877, -0.9495301842689514, 0.020661797374486923, 0.2083931714296341, 0.22650769352912903, 1.6784619092941284, -1.5058174133300781, 0.039985015988349915, 1.9311424493789673, 1.6957467794418335, -1.2383778095245361, -0.32670605182647705, -0.7205557823181152, -1.5032926797866821, -1.4461036920547485, -0.8331480622291565, 0.23989678919315338, -1.8938343524932861, 1.2324575185775757, -0.6942697167396545, 0.3410382568836212, 1.932364583015442, -0.5845600366592407, -1.1656900644302368, -1.6102181673049927, 1.930446743965149, -0.30417004227638245, 0.7258777618408203, -0.46592262387275696, 0.32740503549575806, -0.8699591159820557, -0.2713511884212494, -0.7447600364685059, -0.3378017544746399, -0.7810521721839905, 0.09463447332382202, 1.2210805416107178, 0.8142836093902588, 2.1681461334228516, 0.2388635277748108, -1.9762872457504272, 1.4756836891174316, 0.6942840814590454, -0.9676187038421631, -0.014279579743742943, 0.9743165969848633, 2.002328634262085, -0.583928108215332, -0.8646663427352905, -1.3389368057250977, 0.33027568459510803, -0.5175966024398804, 0.23661848902702332, -0.5029343962669373, -0.02118440344929695, -1.6811543703079224], [-1.3333390951156616, -1.2902253866195679, -0.3929642140865326, 2.570749044418335, -1.6336591243743896, 1.7394073009490967, 0.0006425062310881913, 1.5919413566589355, 1.1111990213394165, 0.27682992815971375, 0.5067181587219238, -0.8335698246955872, -1.4622308015823364, -0.8541417121887207, 1.9282996654510498, 0.37972429394721985, -1.4331709146499634, -0.9680251479148865, 1.6991853713989258, 1.8850152492523193, 0.013029258698225021, 0.11819857358932495, -1.0553574562072754, 0.5472376346588135, 0.1454745978116989, 0.039538491517305374, -0.28069114685058594, -1.257320761680603, 0.6964598298072815, -0.10491462796926498, -0.23395614326000214, -1.6429771184921265, 0.4085608422756195, 0.39006829261779785, -0.7331656813621521, 1.3587312698364258, -0.19484373927116394, 0.2117961049079895, -0.6287140846252441, 1.7443885803222656, 0.21415473520755768, 0.7269644141197205, 1.8151042461395264, -0.05399075523018837, -0.9773785471916199, -1.540437936782837, -0.8722285032272339, -1.9308112859725952, -0.6761055588722229, 0.7441504597663879, -1.508023977279663, 0.8265978693962097, -1.4463740587234497, -0.15512481331825256, 1.0947715044021606, -0.04240138456225395, 0.238884836435318, -0.9678716659545898, -0.2527967691421509, 0.446313738822937, 0.14156563580036163, -1.2593997716903687, 0.32056525349617004, -0.11262395977973938], [-1.3083332777023315, 0.6495218873023987, 0.15293742716312408, 0.11816547811031342, 1.4947526454925537, -0.37029460072517395, 2.008114814758301, 0.27158135175704956, 0.048063214868307114, 0.7206501960754395, -0.04177961125969887, -0.5622061491012573, -0.027535583823919296, -0.17321933805942535, 0.7181575298309326, 1.5261377096176147, -0.26389849185943604, -1.0030407905578613, -0.04848689213395119, -0.21872003376483917, -1.7048799991607666, 1.0244457721710205, 0.7932477593421936, -1.2068142890930176, 0.09200077503919601, -0.6885411143302917, 0.28114885091781616, 0.6752942204475403, -1.8073025941848755, -1.5777333974838257, 0.5840768814086914, 0.7052242159843445, -0.9002032279968262, 0.1946643441915512, -0.009191776625812054, 0.6515457630157471, -0.08330726623535156, -0.7358261346817017, 0.7839617133140564, 0.17356540262699127, 1.1026849746704102, -0.40891727805137634, 0.11959695816040039, -0.9374182224273682, 1.2328100204467773, -1.8940051794052124, 0.0819941908121109, -1.6866334676742554, 0.5776201486587524, 0.5533458590507507, -0.669486403465271, -0.002536401152610779, 0.037496600300073624, -0.41716814041137695, 0.1115884855389595, -1.0914297103881836, -2.108227014541626, -0.3210350275039673, -0.8513522148132324, 1.745098352432251, -0.3739751875400543, 0.20898810029029846, 0.17869932949543, -1.7821370363235474], [-0.39735397696495056, -0.02938893251121044, 0.5542208552360535, 2.238145112991333, 0.692937970161438, -0.05372597277164459, 1.0948673486709595, -0.9216350317001343, 0.4703610837459564, 0.1457081139087677, -0.4243374466896057, -0.1279841512441635, 1.7382317781448364, -0.37646743655204773, 0.20454804599285126, -0.0052711437456309795, 0.5775604844093323, 0.5323547720909119, -1.1322814226150513, 0.4354526698589325, 0.3437919020652771, -0.8870677947998047, -0.6187957525253296, 0.5875118970870972, -0.7454352974891663, -0.06852440536022186, 0.5441735982894897, -0.3162315785884857, 1.818217158317566, -0.017841622233390808, -0.0860663652420044, 0.7277683615684509, 1.3719745874404907, -0.04998583719134331, 0.9166210889816284, -0.5115281939506531, 0.5717926621437073, -1.4149962663650513, 0.35602012276649475, 0.35819482803344727, -0.0342533104121685, -0.38636311888694763, -0.09282573312520981, 0.26104164123535156, 0.8918042182922363, 1.8720051050186157, 0.2767341434955597, 0.7218112945556641, -0.9150763750076294, 0.43937936425209045, 0.06526295840740204, 0.6129964590072632, -0.4749143421649933, -1.1096752882003784, 1.815077304840088, 1.0900850296020508, 0.018573930487036705, -2.1090402603149414, -0.22261519730091095, 0.0006791273481212556, 0.18077029287815094, 0.5314986109733582, 0.7076907157897949, -0.25026628375053406], [-0.8045916557312012, -1.8842271566390991, -0.43377313017845154, -0.8943247199058533, 0.35029202699661255, -0.9201849102973938, -0.20973743498325348, -1.8079476356506348, 0.5574826002120972, -1.4624193906784058, 0.5803983211517334, 0.585699737071991, 1.1947461366653442, -0.33271005749702454, 0.5533355474472046, 2.010843515396118, 0.14527031779289246, 1.0003536939620972, 0.31021007895469666, -0.10207345336675644, -1.7264783382415771, 1.5251320600509644, -0.41611534357070923, 1.8219094276428223, -1.4723972082138062, 0.781273365020752, 0.6420766711235046, 0.0902189314365387, 0.10073975473642349, -0.6163722276687622, -1.4545516967773438, 2.4646244049072266, 1.4052015542984009, -0.2659611701965332, -1.6370899677276611, 0.2536921501159668, 1.5920990705490112, -0.3597334325313568, 0.4381776452064514, -1.7592953443527222, -0.994330108165741, -0.725058913230896, -1.4986376762390137, 0.47566965222358704, -0.1327870786190033, 0.8968674540519714, 1.9753930568695068, -0.6826338171958923, -0.08358927071094513, 0.4981578290462494, -0.7643974423408508, -1.1728379726409912, 0.07348362356424332, 0.19196070730686188, -1.6956062316894531, 0.12770798802375793, -2.2287399768829346, -0.5839188098907471, 0.7081590294837952, 0.8415904641151428, -0.7735862731933594, 0.7370346188545227, -0.5470302104949951, 0.3954763412475586], [-1.1324865818023682, 1.5716619491577148, 0.7694554924964905, -0.1972833126783371, -0.01634226180613041, -1.8078466653823853, 0.45335906744003296, -0.45582708716392517, -0.08402549475431442, 0.04648109897971153, 2.120666980743408, 1.109842300415039, -0.6010242104530334, 0.23951110243797302, 0.3804468512535095, -2.7814252376556396, -0.8011180758476257, 0.6475176811218262, 0.4931000769138336, 0.9993235468864441, -1.2604527473449707, -0.5451884865760803, 0.5231437087059021, 0.2639492154121399, -1.2105263471603394, -0.7405393719673157, -0.09492752701044083, 0.6534978747367859, 0.3471531271934509, 1.5782825946807861, -0.0945083498954773, 0.29517728090286255, -1.2769262790679932, 0.9232704043388367, -0.18332765996456146, 0.04653006047010422, 0.7682851552963257, 0.545989453792572, 0.7154923677444458, -1.2931429147720337, 1.4640978574752808, -0.7219284176826477, 1.033075213432312, 0.7666181921958923, 0.7390316128730774, 0.6044892072677612, 0.3623349070549011, -0.19642722606658936, -0.25965696573257446, -0.3969550132751465, -0.6633504033088684, -0.3451343774795532, -1.627233862876892, 0.23733393847942352, 0.9929364919662476, -0.7839997410774231, -0.9215231537818909, -1.3040024042129517, -1.8463186025619507, 0.5836753249168396, -0.48698627948760986, 0.04695913940668106, -1.759283423423767, 1.7901496887207031], [0.6652159094810486, 0.9054563045501709, -1.3632848262786865, -0.8115912079811096, 0.3560025990009308, -0.5002995133399963, 0.7239171862602234, 0.5757662057876587, 1.0058141946792603, -0.22663503885269165, 0.870149552822113, -0.9265767931938171, 1.5705050230026245, 2.287517786026001, 1.018664836883545, -0.5306094884872437, 0.12343884259462357, -0.2952793538570404, 0.3989964425563812, 0.24353602528572083, 1.6092467308044434, -1.1651194095611572, 0.5150609612464905, 0.21992617845535278, 0.9552311897277832, -1.5211632251739502, 1.7818299531936646, -2.4724478721618652, 0.03340267017483711, -0.8383764028549194, 0.7351166605949402, -0.7015381455421448, 0.16077280044555664, -1.3064557313919067, -0.044731203466653824, 0.5554159283638, 0.38914576172828674, -0.28324446082115173, 1.795637607574463, -1.6533737182617188, -0.6858395338058472, 0.274723619222641, -0.2342483550310135, -1.5352150201797485, -0.34629106521606445, -1.6248036623001099, 0.411359578371048, -0.7680161595344543, -0.39106285572052, -1.721454381942749, -1.1650336980819702, -0.5992630124092102, 0.8728578090667725, -0.8085802793502808, -1.4205620288848877, -0.5376473069190979, 1.352899193763733, 1.5570753812789917, -0.38770464062690735, 1.284096360206604, 1.1534632444381714, -1.1390178203582764, 0.6424070000648499, 1.1494591236114502], [1.4561882019042969, 0.2643531262874603, -0.0701005756855011, 0.2955300509929657, -0.11994252353906631, 1.0531120300292969, -0.3732447922229767, -0.026372205466032028, 0.3451131582260132, -0.916111946105957, -0.5595105290412903, 0.21574798226356506, 0.13646076619625092, -0.852252721786499, -1.9297380447387695, -1.4468867778778076, 0.26029661297798157, -0.1264246553182602, -0.6168134808540344, 0.06209571287035942, -1.7315735816955566, -0.12674884498119354, 0.7234352231025696, 0.4094787836074829, 0.38938504457473755, 0.05602944269776344, -0.1453092098236084, -0.35305148363113403, -0.5484811067581177, 0.39567866921424866, -1.793875813484192, -1.7818366289138794, -0.09702549874782562, -0.7676879167556763, 1.9420045614242554, 1.1627014875411987, 0.21763378381729126, 0.060925256460905075, -1.7534334659576416, -1.5915822982788086, -0.8936445116996765, 0.7708600759506226, 0.1968446671962738, 0.7193440198898315, 0.06811609864234924, 0.7763517498970032, 0.9755931496620178, -0.21598581969738007, 0.6376509070396423, -1.883978247642517, 0.872162938117981, -0.8924828767776489, -0.8019199967384338, -0.11103237420320511, -0.6840009093284607, 0.3547770380973816, -1.4274871349334717, 0.9079747796058655, 0.22760009765625, -0.06146205589175224, -0.03060694970190525, -0.9798767566680908, 0.16059161722660065, -0.21795684099197388], [1.1339772939682007, 0.6772281527519226, 0.9451150298118591, 1.035826325416565, -0.0008701533079147339, 0.2720678150653839, -1.6760327816009521, 0.43850940465927124, 1.2991950511932373, 1.3670780658721924, 0.355234831571579, 0.09262093901634216, 1.5299543142318726, 0.14060044288635254, -1.390775203704834, 0.05693463608622551, -1.1768693923950195, -0.4155004620552063, 0.704460084438324, -0.6010005474090576, 0.174397274851799, 0.6402075290679932, 0.8531313538551331, -0.2352367788553238, -0.35565420985221863, 0.8712440133094788, 1.6902308464050293, -1.2269421815872192, -1.6491801738739014, 0.2599267065525055, -0.07695040851831436, 0.8878954648971558, -0.8574252724647522, -0.6851561069488525, 1.4118998050689697, -1.3721752166748047, -0.8366134166717529, 0.7194199562072754, 0.7535651922225952, -0.4970264732837677, -0.5888087153434753, -1.4315158128738403, 0.1607116162776947, -0.6791881918907166, 0.47149524092674255, 0.18249128758907318, -2.0727906227111816, -0.3241727650165558, -1.993770956993103, 1.4756075143814087, 0.07791386544704437, 0.20294393599033356, -0.5267860293388367, -0.03790314868092537, 0.639726996421814, -0.6496464610099792, 1.076562523841858, 0.5263563990592957, 1.8004120588302612, 0.35245341062545776, 1.50009024143219, 1.6698521375656128, 0.8023168444633484, 0.5128716826438904], [1.4539536237716675, 0.9149024486541748, -0.9472006559371948, -0.02533149905502796, -0.4326372742652893, 0.35727739334106445, 0.6799312829971313, 0.47504645586013794, 1.4705336093902588, 1.210198998451233, -0.6520727872848511, -1.4349277019500732, 0.43579307198524475, -1.9591286182403564, -0.4062960743904114, -0.44068634510040283, 0.2228459119796753, 0.06332462280988693, -0.07720060646533966, 2.9767072200775146, -0.5851355791091919, -0.290188729763031, 1.350714087486267, 0.46601542830467224, 0.3114314377307892, -0.0422569140791893, -1.4024178981781006, 0.053019873797893524, 0.41416117548942566, 1.163525938987732, 1.3560888767242432, 0.23936299979686737, -0.6545792818069458, -0.9235248565673828, 0.7059462070465088, 1.189387321472168, -0.0008880148525349796, 0.33637967705726624, 0.6714540123939514, -0.2825442850589752, 0.24913690984249115, 0.6777125597000122, 0.10432203114032745, 0.07974760979413986, -0.31061092019081116, -0.7376080751419067, -0.5475263595581055, 0.9521651268005371, 1.0627766847610474, -1.435872197151184, -0.730137288570404, -1.1451282501220703, 2.167616844177246, -2.2246315479278564, -0.41818106174468994, 0.5523219108581543, -0.13253897428512573, 0.3016347289085388, -0.5966551899909973, -1.2477504014968872, 1.4849120378494263, -0.3281608819961548, -1.7900034189224243, 0.7824485898017883], [-1.056251883506775, 2.715009927749634, -0.6738789081573486, -0.47631707787513733, 1.3843215703964233, -1.740935206413269, -1.2782936096191406, 0.468110591173172, 1.1271766424179077, -0.8653950691223145, -0.38502606749534607, 1.1504013538360596, 0.16126544773578644, 1.6019443273544312, -0.6559335589408875, -2.2287213802337646, 0.17883528769016266, -0.5072218775749207, -0.5696470737457275, -0.013361281715333462, -0.32818204164505005, -1.479091763496399, 0.18003928661346436, -0.9057676196098328, 0.2867017388343811, 1.7032240629196167, 0.048141106963157654, -0.773095428943634, -0.6384793519973755, 0.6617262959480286, -0.4596630036830902, 0.7767514586448669, -1.3010940551757812, 0.26629775762557983, -1.099075436592102, 0.37281253933906555, 1.5965944528579712, -0.8419760465621948, -1.2307199239730835, -2.2409181594848633, -1.7995285987854004, 0.8044866919517517, 1.7904188632965088, 0.2963297367095947, -0.20097769796848297, 0.3913481533527374, -1.8949881792068481, -1.295061469078064, 1.2153784036636353, 1.670656681060791, 1.2344993352890015, 1.1001204252243042, -0.31956207752227783, -0.2926246225833893, 0.913890540599823, 0.175750732421875, -1.4128053188323975, -1.2682573795318604, -0.43554845452308655, -0.4655711352825165, -2.461264133453369, -0.9540699124336243, -1.0862178802490234, 2.354057788848877], [0.7911161780357361, -0.18911488354206085, 0.5295031070709229, 0.7022464871406555, -1.063346028327942, -0.2902396619319916, 0.9408233761787415, -0.33824360370635986, 0.9886799454689026, -0.39150092005729675, -1.4977378845214844, -1.201197624206543, 0.37908604741096497, -0.8010980486869812, -1.8342268466949463, 1.1680524349212646, 0.49029019474983215, -0.29485899209976196, 0.10461270809173584, 0.16237866878509521, 1.059125304222107, -0.1852564960718155, -0.32558220624923706, -0.6059207320213318, -1.3385778665542603, 1.168319821357727, 0.5136449933052063, -1.0347868204116821, 0.08893007785081863, 1.8261617422103882, 0.02605321817100048, -1.8361042737960815, -0.8763760328292847, 0.32399782538414, -0.6092226505279541, 2.5661091804504395, -0.8923203349113464, 1.0909136533737183, -1.4964535236358643, -0.028698071837425232, -1.917396903038025, 0.19500231742858887, 0.8802657723426819, 2.286655902862549, -0.5637630224227905, -0.860026478767395, 2.290419578552246, -1.3366048336029053, 0.6060933470726013, -0.45804670453071594, 0.6944952011108398, -0.40019795298576355, -0.7259854674339294, 1.6470824480056763, 0.9808973073959351, 0.35987141728401184, 0.634909451007843, 1.159855842590332, -1.7151564359664917, -0.4864749610424042, 2.0358119010925293, 0.4449416697025299, 0.23660361766815186, 0.9641693830490112], [-0.8949785828590393, -0.021622074767947197, -0.6250436305999756, -1.4970968961715698, 0.023246901109814644, -0.1862165480852127, -0.0836542621254921, -0.8594025373458862, -0.35475674271583557, 0.2104513794183731, 0.3905942142009735, -0.10405085980892181, -0.7770746350288391, -2.0903213024139404, 1.2172858715057373, 1.126493215560913, 1.263810157775879, 1.9913673400878906, -0.6460287570953369, 0.32505732774734497, 0.5139806270599365, 1.092822790145874, 0.8573914766311646, -2.082169771194458, 2.5408895015716553, -1.9622678756713867, -1.105880856513977, -1.093604564666748, -1.3543922901153564, 1.2798691987991333, -1.3203800916671753, 0.833282470703125, 1.2961102724075317, -0.8033930659294128, 0.6834202408790588, 0.09541796892881393, -0.3566247820854187, 0.5684022903442383, 1.9051287174224854, 1.628310203552246, 0.777316689491272, 1.5158122777938843, -1.045634388923645, -0.7125810980796814, -0.1372985541820526, 1.4785218238830566, 1.0346379280090332, -0.6090080738067627, 0.9513916373252869, -0.7098460793495178, -0.14820507168769836, -0.8925225734710693, 1.3846189975738525, 0.5434086918830872, 1.3264577388763428, 1.8468644618988037, -0.8015012741088867, -0.5072871446609497, 0.482855886220932, 1.6346155405044556, 0.24837641417980194, 0.31863921880722046, 1.6905169486999512, 0.4909053146839142], [-0.4516226649284363, -1.4856975078582764, -0.6821410655975342, 1.5835415124893188, -0.9372066855430603, 0.027705803513526917, -0.4190330505371094, -0.9827384352684021, -0.774921715259552, -0.42755913734436035, -0.7206875681877136, -0.09204244613647461, -0.36505186557769775, -0.18063661456108093, -1.0848760604858398, 0.6003469824790955, 1.3126674890518188, 1.247290015220642, 0.4778834879398346, 0.5856991410255432, -0.18018269538879395, 0.44917574524879456, 0.39473557472229004, 2.0602426528930664, 0.9969490766525269, 1.897719383239746, 0.08232580870389938, 0.6948878169059753, -0.23620052635669708, 0.7486191987991333, -0.8879519104957581, -0.28428560495376587, -0.3761943578720093, 1.2402175664901733, 0.9097046852111816, -0.8083577156066895, -1.5546671152114868, -1.4451653957366943, -1.5479750633239746, -1.0493708848953247, -0.05739971995353699, -2.1648457050323486, -0.7947807908058167, 0.5466724634170532, -0.3476933240890503, 0.4830033481121063, -1.2420518398284912, -0.22694526612758636, -1.2692341804504395, 0.622701108455658, -0.12353511899709702, 0.9662380814552307, 2.120690107345581, 0.5718235969543457, -0.09960618615150452, -0.5364685654640198, 0.5887389779090881, -0.21363204717636108, 0.30687063932418823, -0.12352283298969269, -1.0416336059570312, 1.2725752592086792, -0.5805290937423706, 1.737317442893982], [-0.6497963666915894, -1.950724720954895, -2.2817208766937256, -0.32264187932014465, 0.9248577952384949, -0.567298173904419, 0.6543440222740173, 0.8613268733024597, -0.16385002434253693, -0.6934324502944946, -0.9912139773368835, 1.0160720348358154, -0.08129566162824631, 1.5215682983398438, -0.6269931197166443, -1.1550288200378418, -0.008503246121108532, -1.5431057214736938, -0.8341385126113892, 0.7008342742919922, -1.4243223667144775, -0.6582763195037842, 0.6181714534759521, -0.3481481969356537, -0.28352558612823486, 0.5649526119232178, 0.9816697835922241, -1.7644944190979004, -1.9819087982177734, -0.1722264587879181, 0.9749840497970581, -0.0592336431145668, 1.7277532815933228, -1.221856951713562, -0.8501419425010681, 0.7004331946372986, -1.6129560470581055, -1.531926155090332, -0.08689969778060913, 1.2606379985809326, 1.2954254150390625, -0.5129035115242004, -1.4835411310195923, 1.2021758556365967, -0.5181654095649719, -0.24392879009246826, -0.4374927878379822, -0.7660645246505737, 0.1923568844795227, 1.407732367515564, 0.2779451608657837, 0.9674700498580933, 0.30206888914108276, -1.5478419065475464, -1.321216344833374, -1.1165639162063599, -0.6174235343933105, -1.396082878112793, -1.1354255676269531, 0.5258146524429321, 0.4736382067203522, 2.1303858757019043, -8.592647645855322e-05, 0.09621381759643555], [-0.9206749200820923, -0.29720795154571533, 1.2495509386062622, -1.3327146768569946, -0.025577036663889885, -1.1830387115478516, -0.14218497276306152, 2.703953504562378, 0.8653303980827332, -1.2428325414657593, -3.530674934387207, 0.7871127724647522, -2.0336406230926514, -0.2174457162618637, -1.4143022298812866, 0.8015451431274414, -0.3278248608112335, -0.6828787326812744, 0.6756470799446106, 0.9050363302230835, 0.24029965698719025, -1.6560280323028564, -0.6280542612075806, 1.032069444656372, -2.032174825668335, -2.554368257522583, 1.1877694129943848, -0.24979421496391296, 0.8272569179534912, 0.8247644305229187, -0.12557953596115112, -0.6605157256126404, -0.40657639503479004, 0.504544734954834, 1.0560497045516968, 0.971483051776886, -0.26958167552948, 0.4052882790565491, 0.13938476145267487, 0.3983602523803711, 0.7721923589706421, 0.172966867685318, -2.2213528156280518, 1.0340726375579834, -1.9047489166259766, -0.34720271825790405, -0.8117367625236511, -0.8651323318481445, 0.25481829047203064, -0.27195408940315247, 0.2856077551841736, 1.185281753540039, 1.8842365741729736, 0.841802179813385, -0.4425356686115265, -1.1001813411712646, 0.030889147892594337, -0.24528585374355316, 1.7895464897155762, -0.04475386068224907, -0.47572433948516846, 0.7109814286231995, 0.5187169909477234, 0.523276150226593], [0.214812770485878, -0.085037961602211, -0.25106939673423767, -0.6110848188400269, -0.7887175679206848, 1.3310303688049316, -0.9404072165489197, 1.6125375032424927, 1.5788854360580444, 0.44927671551704407, 2.190289258956909, 0.6982913613319397, 0.66921067237854, 0.28603947162628174, -0.8470760583877563, -0.5645437240600586, 1.2753812074661255, -0.0947272852063179, 0.042858682572841644, 1.3799954652786255, -1.1651625633239746, 0.4148823320865631, -0.2427898347377777, -1.7691599130630493, -0.4121306538581848, -0.7446964979171753, 0.5572453141212463, 1.4937561750411987, -0.4605744481086731, 0.6724761724472046, -0.2837729752063751, -1.8798903226852417, -0.5959212183952332, -0.45180264115333557, 0.04505826532840729, -1.1848684549331665, 0.13150593638420105, 1.431738018989563, -0.04018506035208702, -0.7451223731040955, -0.44001927971839905, -0.3446762263774872, 1.3133249282836914, -2.0763463973999023, 0.9258373379707336, -1.7236051559448242, 0.10541445016860962, 0.775031566619873, 0.4944700598716736, 0.22819599509239197, -2.3128645420074463, 0.9645535349845886, -0.16320963203907013, -1.2443712949752808, 0.4039890766143799, 0.3200203776359558, -1.770538091659546, -0.3273465633392334, 0.7778356671333313, -0.6189680695533752, 1.2085115909576416, -0.30282098054885864, -0.6549845933914185, 1.556477427482605], [-0.7706227898597717, -0.9951931834220886, 1.5920151472091675, -1.4915082454681396, 0.5687689185142517, 0.12980413436889648, 0.3606143593788147, -0.21078439056873322, 0.6072463393211365, 0.3021511137485504, -1.9390246868133545, 1.0484991073608398, 1.942674994468689, -0.5366895794868469, -1.2434892654418945, 1.3834290504455566, 2.853494644165039, -0.19516658782958984, 1.7951256036758423, -1.5975964069366455, -0.7626796364784241, 0.11851431429386139, 0.4135276973247528, -0.3286178708076477, 1.5460203886032104, -1.089913010597229, -1.5859414339065552, 0.5719178915023804, -3.80580472946167, -0.4879050850868225, 2.1824076175689697, 0.6362631916999817, -0.29337096214294434, -0.3529582619667053, 0.33700308203697205, -1.839380145072937, 0.13288739323616028, -0.4982009828090668, -0.8148212432861328, 1.349876880645752, 1.5060863494873047, 1.251549482345581, 0.7509818077087402, -0.42463064193725586, 1.2661826610565186, -0.5066109299659729, 0.8077986836433411, 0.44528862833976746, 0.2769685983657837, 0.49688467383384705, -0.32791054248809814, -0.5157872438430786, 0.4538787305355072, 3.45133638381958, -1.9209458827972412, 0.2332136332988739, -0.4189273416996002, 1.9516210556030273, -0.1905197650194168, 0.1843816488981247, -0.5588135719299316, -0.7804850935935974, 0.054800596088171005, 0.6071887612342834], [2.5575358867645264, -1.1104167699813843, 0.02601362206041813, 0.33954381942749023, -2.1358072757720947, -0.5487971305847168, 0.5665295124053955, -0.9484941363334656, -0.03839820250868797, -1.6920661926269531, 0.2665756642818451, 0.6487910747528076, 1.4368919134140015, 0.6639547348022461, 0.548850953578949, -0.9885948300361633, 0.5211122632026672, -0.604087769985199, 0.40809932351112366, -0.5244901776313782, 1.5351226329803467, -0.4865182042121887, 0.3521769642829895, 0.792037844657898, -0.1883811354637146, 0.7228943705558777, 0.9232257604598999, -0.35207125544548035, 1.2503231763839722, 0.7254006266593933, 0.3977735936641693, 1.7172911167144775, -1.0757646560668945, -1.6008752584457397, 1.0920180082321167, -0.0895865336060524, -1.2603412866592407, -0.6804468631744385, -1.0599290132522583, -1.2259001731872559, -0.667400598526001, 1.354780673980713, 1.4600337743759155, 1.0158876180648804, -0.4126964807510376, -0.19288212060928345, -0.5791076421737671, 0.28790080547332764, -1.0921764373779297, 0.34338197112083435, -1.3537729978561401, 2.0307133197784424, 1.9258885383605957, 0.05696342512965202, -0.30234360694885254, 0.11478956043720245, 1.8192775249481201, -0.6849105358123779, -0.5205072164535522, 0.2815721929073334, 0.5433743000030518, -0.2827252745628357, -0.1364886313676834, 0.8838830590248108], [-0.8255487084388733, 1.0664035081863403, 1.2824610471725464, 1.5375796556472778, -1.3872569799423218, 0.0048998030833899975, -0.7902052998542786, 0.7611229419708252, -0.1402142196893692, 0.7224688529968262, 0.5742761492729187, 1.004483938217163, -1.7206014394760132, -0.7245892286300659, 2.0183568000793457, 0.5476002097129822, -1.2963701486587524, -0.9165894389152527, -1.3325140476226807, 0.9779730439186096, 0.4701598584651947, -0.05937841907143593, -1.886974811553955, -1.2481380701065063, -0.23313327133655548, 0.011876881122589111, -0.07068034261465073, 0.9186455011367798, -0.6151625514030457, -1.5984545946121216, 0.620190441608429, 0.6201083064079285, -0.37027981877326965, 1.2875369787216187, 1.3761883974075317, 1.5121361017227173, 0.5489659309387207, 0.21966037154197693, 0.2235284298658371, 0.45201361179351807, -0.06323233991861343, 0.8173366785049438, -0.6859645247459412, 0.13918675482273102, -0.25825679302215576, 0.542990505695343, -0.3989700675010681, -2.2324306964874268, 0.9393404126167297, 1.1731536388397217, -0.02682962641119957, -0.9186043739318848, 0.1035841852426529, 0.8277150392532349, -1.0852391719818115, -1.1788723468780518, 1.222862958908081, 0.5545908808708191, 0.2452298253774643, -0.8722712993621826, -0.17501375079154968, 0.3899247944355011, -0.1645936220884323, 0.22532771527767181], [-1.1076610088348389, -0.6555653214454651, 0.4062166213989258, 0.6396507024765015, -0.002457135124132037, 0.2708699703216553, 0.6157020926475525, -0.07098384201526642, -0.320061057806015, 0.9806526899337769, -0.3642941415309906, 0.6231070160865784, -0.12393157184123993, 1.5230376720428467, 0.22761081159114838, 0.7088860869407654, -0.2332647144794464, 0.3215799629688263, -0.2200237214565277, -0.6748070120811462, -0.07625125348567963, -0.8203681707382202, -0.9527690410614014, -0.25691673159599304, -0.5611295104026794, -0.32183489203453064, -0.14076606929302216, 0.5075134634971619, 1.1872014999389648, -0.3369399309158325, 0.17185866832733154, 0.727771520614624, 1.398176670074463, -0.33182111382484436, 0.4848310947418213, 0.8847615718841553, -0.3493902385234833, 1.1764910221099854, -2.214808702468872, -0.49905937910079956, -1.5817514657974243, -1.7812272310256958, 0.13371065258979797, -0.17113132774829865, 0.594581127166748, -0.08141879737377167, 1.6439173221588135, 2.2253549098968506, -1.7823023796081543, -0.6062046885490417, -0.09581490606069565, -0.37025031447410583, -0.6248225569725037, 0.5218995809555054, -0.579980731010437, -0.34958046674728394, 0.16866032779216766, 0.5889312028884888, -0.017684530466794968, 1.812971591949463, 1.1524256467819214, 0.18601590394973755, -0.008339165709912777, -0.2510960102081299], [-0.5001533031463623, 1.489441990852356, -1.4196929931640625, -0.35200682282447815, -1.7223358154296875, 0.06029333546757698, 0.0908297598361969, -0.20191025733947754, -0.6855600476264954, -0.3163701295852661, -0.04271091893315315, 0.5145991444587708, 1.972924828529358, -0.36145687103271484, -1.2953215837478638, 1.0808435678482056, -0.9827097058296204, 0.002163682598620653, -0.16895133256912231, 0.7857654690742493, -0.8555324673652649, -0.3317449688911438, 0.270988792181015, -1.7526787519454956, 0.8176233768463135, 1.041881799697876, 2.0465176105499268, -0.40918540954589844, -0.8616999387741089, 0.6815145611763, 0.2980089783668518, 0.24725502729415894, -0.3276353180408478, 0.27685222029685974, 0.6312452554702759, 2.342480421066284, 0.2681483030319214, 1.0809730291366577, 0.8654306530952454, 0.8593370318412781, 0.15044201910495758, 0.2976607084274292, 1.3140863180160522, 0.5732330083847046, 0.137738898396492, 1.2319892644882202, -0.31494301557540894, 0.8284561634063721, 0.2686193883419037, -1.0412566661834717, -1.1987003087997437, 0.497104674577713, 0.19859249889850616, -0.1831265389919281, 0.05576855689287186, 1.7633599042892456, -1.293270230293274, 1.691210150718689, -0.33504730463027954, -1.4315487146377563, -0.35365673899650574, 0.18638506531715393, 0.7398566603660583, -0.6821608543395996], [-1.593136191368103, -0.03500889614224434, 0.1438320279121399, 1.450863242149353, -0.979275643825531, 0.3619001507759094, 0.5829927325248718, -1.8445186614990234, -1.0844084024429321, 0.27218809723854065, -0.4060298204421997, 0.4534653425216675, 1.231865406036377, -0.1815125197172165, -1.1557703018188477, -1.6264488697052002, 0.3976946771144867, 1.4504382610321045, 1.9910451173782349, -0.5186271667480469, 0.1820070594549179, 0.21491487324237823, -0.2978098392486572, -1.1975164413452148, -0.17156878113746643, 0.23942112922668457, -0.22021272778511047, 0.16241782903671265, 0.05630607157945633, -0.4575631320476532, 1.642532229423523, -0.7220540642738342, -0.35078975558280945, -0.4661270081996918, 0.45291370153427124, -0.5349478125572205, 0.1935373842716217, 0.4828244149684906, 1.4742920398712158, 1.0882285833358765, -0.15280519425868988, -1.7994235754013062, -3.0298213958740234, -0.7478132247924805, 0.09805559366941452, -2.035757064819336, 0.10555189847946167, 0.01282463688403368, -0.44852596521377563, 0.814662516117096, -0.17511051893234253, -0.11304714530706406, -1.3967621326446533, -1.138352870941162, -0.509261965751648, 0.827504575252533, 0.5244652032852173, 0.9321315288543701, -0.2778175473213196, 2.076533317565918, 0.6763202548027039, 2.6449480056762695, 0.03168206289410591, -0.3852890133857727], [-1.2395548820495605, 0.3285464644432068, -1.4620723724365234, -0.13421490788459778, 0.1746508777141571, -1.1948968172073364, -0.9564350247383118, -1.771925449371338, -0.7295148968696594, 0.05372583866119385, 1.2246965169906616, 0.7787209153175354, 0.9872470498085022, -1.5865824222564697, 0.20126739144325256, -1.0709495544433594, -0.07374069839715958, 0.29804515838623047, -0.42932945489883423, 0.4055456817150116, 0.9132986664772034, 0.6227635145187378, -1.1260188817977905, -1.1286706924438477, 1.17603600025177, 0.8955855965614319, 0.21004703640937805, -1.7838895320892334, -0.4336736500263214, 2.7397994995117188, 0.6695424914360046, -1.5502997636795044, -1.4723541736602783, -1.3139152526855469, -1.3037223815917969, -0.44610482454299927, 1.6586062908172607, 0.8453873991966248, 0.07845791429281235, -0.3365510106086731, -1.366443157196045, -1.1260898113250732, 0.5018501281738281, 1.7048828601837158, 1.0993757247924805, -0.28063255548477173, -0.6158408522605896, 0.013462993316352367, -1.2268836498260498, 0.3797207474708557, -0.27979570627212524, -0.8929638862609863, -1.0937522649765015, 0.00902189128100872, 0.05385682359337807, 1.9459925889968872, -1.0775700807571411, 0.07431832700967789, 0.2401009351015091, -0.16948722302913666, 0.21716570854187012, -0.21125593781471252, -0.5895723104476929, -0.05130773410201073], [3.0253376960754395, 1.5898948907852173, -0.8812888264656067, 1.0976389646530151, -1.8349189758300781, 1.6738359928131104, -0.9369019865989685, 1.3864901065826416, 0.35563531517982483, -1.157346487045288, 0.7847741842269897, -1.4810434579849243, 0.2518503665924072, 1.6389023065567017, -1.7606201171875, -0.011778981424868107, 0.37332460284233093, 1.031119704246521, 2.356478452682495, -0.6499766707420349, 0.6325485110282898, -1.5663447380065918, -0.9324867129325867, -0.04329516738653183, -0.19216878712177277, 1.179086685180664, 0.6518532633781433, -0.5826380252838135, 0.2351338267326355, -0.8775812983512878, -0.2804148197174072, -0.3229631781578064, 0.4629261791706085, -2.6730151176452637, 1.9691221714019775, 0.41716268658638, -1.7705763578414917, -0.5322444438934326, -0.015289943665266037, 0.8178749084472656, -0.47222062945365906, -1.1204472780227661, -1.2792699337005615, -0.35308873653411865, -1.180633783340454, 1.2048451900482178, -0.17788216471672058, 0.21171772480010986, -0.9212720394134521, 2.2125489711761475, -0.5796451568603516, -0.7387146353721619, 1.7659692764282227, -0.59102863073349, 0.5423975586891174, 0.5436680316925049, 0.32422661781311035, -1.740206241607666, 0.4660783112049103, -0.5128495097160339, 2.2719695568084717, -1.8933988809585571, 0.1503610461950302, -1.930046796798706], [1.3880749940872192, -0.7656837105751038, 1.0544536113739014, 0.5616397261619568, 0.6978131532669067, -0.2341252565383911, 0.029622556641697884, 1.135204792022705, -0.018624017015099525, -0.939243495464325, 1.8200465440750122, -0.4965376555919647, -0.48485273122787476, 1.0887508392333984, 1.023885726928711, 1.0522127151489258, 0.6751818656921387, 0.5929131507873535, 1.1826395988464355, 1.8884093761444092, -0.3583749532699585, -0.16269610822200775, -1.350034236907959, -0.6432151198387146, 0.568082332611084, 0.22203464806079865, -0.16938932240009308, 0.35463374853134155, -2.112325668334961, -0.15432128310203552, -1.0337122678756714, -0.41791462898254395, -0.15440598130226135, -2.0038421154022217, -0.9964045882225037, -0.4283839762210846, 0.00394462700933218, 0.5212889313697815, 0.808070957660675, 0.009039920754730701, 0.6892815828323364, 0.3527131676673889, 0.9240713715553284, -0.4876890480518341, 0.4371691942214966, -0.3782041668891907, -0.8958970904350281, -0.09310904145240784, -1.4936188459396362, -0.25912636518478394, -0.6541671752929688, -0.3436393141746521, -0.40100720524787903, -1.0907179117202759, 0.5582043528556824, 0.09150227904319763, -2.0334255695343018, -0.8959141969680786, -0.45301932096481323, 1.2239668369293213, -1.3472025394439697, -0.5206704139709473, -0.5403875708580017, 0.2048500031232834], [0.27994054555892944, -0.9317725896835327, -0.7594104409217834, -0.5492115020751953, -0.6883296966552734, 0.8308882713317871, -1.1596965789794922, -1.3428738117218018, 1.1494086980819702, 1.3519309759140015, -0.45938840508461, 0.49495387077331543, -0.8963717818260193, 1.958585262298584, -1.1620866060256958, -0.7864134311676025, -0.7055685520172119, -0.6751287579536438, 1.320719599723816, 0.5692773461341858, 0.834230899810791, -0.7214467525482178, 1.5326952934265137, -1.0437402725219727, 1.4725689888000488, -0.9955326318740845, -0.7622725963592529, -0.5125476121902466, 1.6662033796310425, -1.0876736640930176, 2.5733492374420166, 0.20148035883903503, 1.0776787996292114, -1.285622000694275, -2.530827760696411, -0.5043826699256897, 0.7323295474052429, -1.1805634498596191, -0.09245085716247559, 0.3477412164211273, 0.4480994641780853, -1.1020126342773438, 0.6715113520622253, 0.21736489236354828, -0.41158515214920044, 0.07457458227872849, -1.391711950302124, -0.2843720614910126, -0.21854615211486816, 0.4162614047527313, -0.752957820892334, -1.4254249334335327, -0.037191241979599, 2.8202409744262695, -0.8626483678817749, -0.25702929496765137, -0.249464750289917, 0.2738524079322815, -0.6638110280036926, 0.6306988596916199, 0.3941073715686798, 0.34137994050979614, 0.7544434666633606, 1.4007580280303955], [0.2751062214374542, -0.8752241730690002, 1.4193425178527832, -1.1661828756332397, 0.07070138305425644, -0.6690141558647156, -0.08660300076007843, 0.5609278082847595, 2.6945178508758545, 1.094774842262268, -1.8779990673065186, -0.048016276210546494, 1.082927942276001, 0.4404067099094391, -1.2676308155059814, 0.2851555645465851, 1.1313648223876953, -0.8946636319160461, -1.18646240234375, 0.3599866032600403, 0.363607794046402, 0.9259792566299438, 0.31909725069999695, -1.4935511350631714, 1.690995454788208, -1.251725435256958, -0.5850920081138611, 1.3979469537734985, -0.21823103725910187, -1.3020299673080444, 0.8107182383537292, -0.4630073606967926, 1.02885103225708, -0.6495471596717834, 1.3895249366760254, -2.735128164291382, 0.012562333606183529, -2.328585386276245, 0.5921863317489624, 0.7926493883132935, 0.9431992173194885, 0.8544673919677734, 0.991835355758667, 0.3058427572250366, 0.023155774921178818, 0.8018847703933716, -0.8684500455856323, 0.8262579441070557, -0.5484486818313599, -0.030628304928541183, 0.584212064743042, -1.7609531879425049, 0.500651478767395, 1.2760038375854492, 0.09447477757930756, 0.9780438542366028, 0.30810049176216125, -1.152012586593628, -1.3754247426986694, 0.05657454952597618, -1.2422003746032715, -0.008606234565377235, 1.0085110664367676, -0.2774524390697479], [0.8329749703407288, -0.43420514464378357, -0.634247899055481, 0.3975342810153961, 2.0369443893432617, -1.3616141080856323, 0.589180588722229, 0.650978684425354, -0.10667769610881805, 0.5745212435722351, -0.7809728980064392, 0.5677770376205444, 2.2874197959899902, 1.2485102415084839, 1.599632978439331, -0.13054899871349335, -0.22084401547908783, -1.7758382558822632, -1.766487717628479, 0.8180784583091736, -0.6676297187805176, 0.4273722767829895, 1.970367670059204, 1.7024208307266235, -0.40768927335739136, -1.3289554119110107, -0.3356482684612274, 0.16305413842201233, 0.8012586832046509, -0.13475416600704193, 2.2676892280578613, -0.7098914980888367, 1.9873682260513306, -3.4404900074005127, 1.0146121978759766, 1.4376906156539917, 0.42058953642845154, -0.26688963174819946, 0.1951419562101364, 0.1630958467721939, 1.1028571128845215, 1.3316309452056885, 0.5705180168151855, 2.9906487464904785, -0.8781279921531677, -0.33128082752227783, -0.5268105268478394, -0.6947612762451172, 0.12339720875024796, 1.1782455444335938, -0.3463731110095978, 0.24551144242286682, 0.11987432092428207, 0.28186920285224915, 0.4162399172782898, -1.148659110069275, 0.2242060899734497, 0.01594296470284462, -1.2297511100769043, 2.075488567352295, 0.5023795366287231, 0.9768792986869812, -0.9018629789352417, -2.3877477645874023], [-1.3773995637893677, 1.532044529914856, 0.7345921397209167, -0.2945846915245056, -0.30321767926216125, -0.04459621012210846, 0.5516239404678345, -0.005974777042865753, -0.14160269498825073, 0.5588753819465637, 0.2057444006204605, 0.4947228729724884, -0.8631414771080017, 0.3754322826862335, 0.28130877017974854, -1.3963847160339355, 0.8943023085594177, 1.6502269506454468, -1.4288158416748047, 0.557659924030304, 0.9249348640441895, 1.029801845550537, 0.3378888964653015, -0.474334716796875, -0.6484590172767639, -0.07548154145479202, 0.9142074584960938, -0.031631577759981155, 2.468751907348633, -0.5957289338111877, 1.5589286088943481, 0.7451454997062683, -0.6755172610282898, -0.3628286123275757, 0.40297016501426697, 0.17655470967292786, 0.8776566982269287, -0.6593489050865173, -0.2539486289024353, 0.9633957743644714, 0.8488543033599854, 1.6126141548156738, 1.2391438484191895, 0.26733922958374023, -1.4673755168914795, 1.2773631811141968, 0.06375836580991745, -0.5437570214271545, -0.34832340478897095, 1.1135783195495605, 0.008744129911065102, -1.4457805156707764, 0.08851253241300583, 0.4744032621383667, 0.08341573178768158, -0.8717641830444336, -2.158907651901245, -0.995433509349823, 0.8518895506858826, -0.5515512824058533, 0.08742381632328033, -0.3057456612586975, 0.7586574554443359, -0.5877099633216858], [-0.6334714889526367, -1.2470016479492188, -0.2768179476261139, -0.050770822912454605, -0.8260273337364197, 1.0483038425445557, 1.1556779146194458, -0.7107570171356201, 0.614109218120575, 0.5994019508361816, -0.01749640889465809, 0.27671387791633606, 3.2519774436950684, 0.10085820406675339, 0.18247908353805542, -1.9985551834106445, 1.2417323589324951, 0.15267573297023773, 0.789804995059967, -1.295873761177063, 0.8160449862480164, 0.6800913214683533, -0.27914807200431824, 1.1761622428894043, 1.5982513427734375, -0.18751104176044464, 0.03714048117399216, -2.43967604637146, 0.6087958812713623, 0.4756593108177185, -1.0040727853775024, -0.8992509245872498, 0.3522355556488037, 0.4712386131286621, 0.4811570644378662, -0.024751536548137665, 1.746504545211792, -0.08742565661668777, 0.5608022809028625, -0.8482595086097717, 0.33513495326042175, 0.25596630573272705, -1.0344659090042114, 1.0050718784332275, 0.4830555021762848, -0.06458637118339539, -1.3250638246536255, -0.5268017053604126, -0.4250562787055969, 1.7933675050735474, -0.18020601570606232, -1.7059473991394043, -0.838746190071106, 0.3088683485984802, 0.3207932412624359, 2.8953099250793457, 1.5136553049087524, 1.057511329650879, -0.4198564887046814, 0.3379521369934082, 0.43304920196533203, -1.2039122581481934, -2.1388401985168457, 0.8317506909370422], [-0.8773427605628967, 0.4901539385318756, -0.3471662104129791, -0.19397497177124023, 2.541181802749634, 0.49999895691871643, 1.7867119312286377, -0.7743534445762634, -0.8420729041099548, 0.11270270496606827, -0.5233561396598816, 0.05943930521607399, -1.0714020729064941, 1.0656681060791016, -1.2766287326812744, -0.2409522980451584, 0.22961540520191193, 1.9132088422775269, -0.41258788108825684, -1.888929009437561, -0.09213168919086456, 1.365740418434143, -0.16461515426635742, 0.23856595158576965, -0.06774075329303741, 1.3161331415176392, 0.4811256527900696, -2.1473851203918457, -0.12677167356014252, -0.9390947818756104, 0.5217241644859314, 0.6109693050384521, -0.19053173065185547, 0.03183495253324509, -0.001353073283098638, -2.7080399990081787, 0.25739771127700806, 0.6947602033615112, 0.4313284456729889, 0.21254850924015045, 0.1117398738861084, -0.9296103715896606, -0.7745314836502075, -0.40211614966392517, -0.5009007453918457, 0.38415032625198364, 1.9141536951065063, 0.3691282272338867, -0.24091999232769012, 0.17888037860393524, 1.7224751710891724, 0.20034272968769073, 0.3463899493217468, -0.2522451877593994, -0.5852857232093811, 0.5591357946395874, -0.5848774909973145, -0.05708054080605507, 0.7891103625297546, 0.14890654385089874, -1.3981446027755737, -3.236659526824951, -0.42267394065856934, -0.37108075618743896], [0.2983447313308716, 1.2633837461471558, 0.9456549286842346, 0.25461968779563904, -0.4730939269065857, 0.12986846268177032, 0.6471060514450073, 0.34185758233070374, -0.13383525609970093, -0.96208256483078, 0.048558201640844345, -1.1270153522491455, -2.884237766265869, 1.185043215751648, -0.2573561668395996, -0.09193883836269379, -2.006115436553955, -0.8879702091217041, -1.3590211868286133, -1.0662785768508911, -1.043818473815918, -3.2644169330596924, 0.49476903676986694, 0.35107916593551636, 0.5311805605888367, -0.43995437026023865, 1.0571213960647583, 0.5197288990020752, -1.475274920463562, -0.48972445726394653, 0.3719295859336853, -1.5381532907485962, -0.03716767951846123, -0.5694352388381958, 0.8752593994140625, 0.6480090022087097, -0.34251800179481506, -0.2903658151626587, -0.0690147653222084, 0.9323695302009583, -0.8152280449867249, -1.8162461519241333, -0.031942255795001984, 0.06810404360294342, -1.369468331336975, -1.797878623008728, -0.5289672017097473, -0.18908242881298065, 0.3955745995044708, -2.9780209064483643, 0.3831847906112671, -0.9839147925376892, 0.2650323808193207, -0.9039272665977478, -0.6659967303276062, 0.026220185682177544, 0.06704013049602509, 0.8722589612007141, 2.934631109237671, 1.5679632425308228, -1.283568263053894, 0.027210446074604988, 0.8706064820289612, 0.2656424343585968], [0.4720029830932617, 1.3929015398025513, 0.7509042620658875, 0.5379635095596313, -0.23342151939868927, -0.8401814103126526, -0.49297434091567993, 0.6255249977111816, 0.376608669757843, 0.30474603176116943, 1.9760890007019043, -0.17666028439998627, 1.3238543272018433, -0.6138001084327698, -0.21814092993736267, 1.3916840553283691, 0.7496208548545837, -0.35024821758270264, -1.144408941268921, 0.003622794058173895, 1.3502182960510254, -0.23885290324687958, 1.0159156322479248, -1.0598831176757812, 0.9168453216552734, 0.12352262437343597, -0.6178640127182007, 0.08655470609664917, -0.38996732234954834, -0.6555100083351135, 0.043210241943597794, 0.43401604890823364, 0.6827839612960815, -1.5765316486358643, 0.6917763948440552, 0.43263211846351624, -1.2536673545837402, -0.3320256769657135, 2.4947257041931152, -1.2192983627319336, 1.1000163555145264, -0.622911810874939, 2.6535685062408447, 0.6417933702468872, 0.5713503360748291, -0.6966535449028015, 1.4489091634750366, -0.02934054099023342, -2.600552797317505, -0.9081827998161316, -0.8284318447113037, 0.25392958521842957, 0.1779385507106781, 0.6697443723678589, 0.44512638449668884, -0.2824952304363251, -0.46552419662475586, 1.8896288871765137, 0.3035643994808197, -0.21900850534439087, -0.3708359897136688, 0.7202669978141785, -0.29371365904808044, -0.34343433380126953], [-0.7656282186508179, -1.2387248277664185, 0.13692685961723328, 2.1603775024414062, 0.3943789601325989, 0.7072067856788635, -0.2656556963920593, 2.807731866836548, -1.8700472116470337, -1.247055172920227, -0.8962850570678711, -0.31375813484191895, -0.40401697158813477, 0.5038228034973145, 0.06456095725297928, -1.234355092048645, -0.6691901683807373, -0.3458818793296814, -0.6018064022064209, -0.24173352122306824, -0.7584022879600525, -0.6147647500038147, -0.4375819265842438, 0.6562528610229492, -2.1331706047058105, 0.8059780597686768, 1.0950931310653687, 0.9694651961326599, -0.9256632924079895, 0.6334653496742249, -0.6886805891990662, 1.2993857860565186, -0.18209344148635864, 0.6486589908599854, -1.1435223817825317, -0.7763862609863281, 0.2771739959716797, 0.2793910503387451, -1.004124641418457, -0.5664456486701965, 0.7652749419212341, -0.019452938809990883, 0.9640042781829834, -1.3444989919662476, -0.8233258724212646, -0.7493588328361511, -1.3951181173324585, 0.24837476015090942, -0.6368250846862793, -0.36716848611831665, 2.038935661315918, 0.053268913179636, 0.772155225276947, 1.2985343933105469, 0.33517998456954956, 0.30416110157966614, -0.7585071921348572, -0.6580480933189392, 0.058177608996629715, 1.2456868886947632, 0.9714586138725281, -0.6253857016563416, 0.272164523601532, -1.865344762802124], [2.4780287742614746, -0.06183728203177452, 1.079399824142456, -0.15234051644802094, -0.2387138456106186, 0.743054211139679, 0.03755921125411987, 0.5832957625389099, 0.9424247741699219, 0.6175352334976196, 0.3666273355484009, -0.09035459160804749, 0.44546008110046387, -1.469763994216919, -0.19523657858371735, 1.380185842514038, 1.1436021327972412, 0.33020102977752686, -0.4581722915172577, 0.3383292555809021, 0.4699456989765167, -0.29168280959129333, -1.1399500370025635, -1.6821972131729126, -0.35525548458099365, -0.4903242290019989, -1.0670429468154907, 2.1625726222991943, 0.3971828520298004, -0.1825246512889862, 0.6826568841934204, 0.1927863508462906, -0.3565729856491089, -0.2976512014865875, -0.3345758020877838, 0.7097728848457336, 0.027800031006336212, 1.9442416429519653, -0.701564371585846, 1.1456127166748047, 0.9127652645111084, 1.4400595426559448, -0.08546143770217896, -0.8731671571731567, -0.2604043781757355, 0.26938241720199585, 2.7910618782043457, -0.96267169713974, -0.8332553505897522, 0.3680519163608551, 2.070528268814087, -0.10126709938049316, -1.7710739374160767, 1.0796705484390259, -0.586489200592041, -0.2818605601787567, -1.1254693269729614, -1.287623405456543, 0.7325845956802368, 2.178241729736328, 0.6494305729866028, 0.39645105600357056, -0.3446178734302521, 0.28017812967300415], [-0.04497556388378143, -0.7963460683822632, -0.23357675969600677, 0.5812296867370605, 1.124502182006836, -0.5488139986991882, 0.09966090321540833, 1.7284724712371826, -0.25146251916885376, -0.9694119691848755, -0.8705052733421326, 0.5669983625411987, 1.8226523399353027, 0.09214954078197479, 0.6943396925926208, -0.897779107093811, -2.218522071838379, -0.7360557317733765, -0.7423680424690247, 0.1705654263496399, 0.3764338195323944, 1.3257756233215332, 0.34322720766067505, -1.0446363687515259, -0.15518228709697723, -0.15248319506645203, -0.6660963296890259, 2.2135729789733887, 1.144312858581543, -1.5314832925796509, 0.508898913860321, 0.9613644480705261, 1.715200662612915, -0.7538236379623413, -0.039431508630514145, -0.1161169484257698, -0.018498620018363, 0.9621596932411194, 0.2913144826889038, 0.25011301040649414, -1.2946739196777344, 1.4210586547851562, 0.8146181106567383, 0.4491499364376068, -1.5608068704605103, -0.24853456020355225, 2.410890579223633, 0.09462888538837433, 0.20791782438755035, -0.4805871844291687, 1.0777186155319214, 1.638831615447998, 1.2344001531600952, -0.2150331288576126, -0.6396616697311401, 0.28400447964668274, -1.118635654449463, -0.8639245629310608, -0.43285226821899414, 1.2146081924438477, 0.002004266018047929, 0.8701013326644897, -1.8387722969055176, 1.3291879892349243], [-1.3818295001983643, 0.3656473159790039, -1.283686637878418, 0.7553377747535706, -1.0623832941055298, -1.5322080850601196, 0.13256923854351044, -0.30838990211486816, -0.09998225420713425, 1.2744978666305542, -1.6975765228271484, 0.9044251441955566, 0.43597760796546936, -0.08205267786979675, 0.7477833032608032, 1.242571473121643, -0.2932656705379486, -0.245614156126976, -0.8877925872802734, 0.8301988244056702, -1.9734441041946411, 1.855094313621521, -0.15062318742275238, -0.09509502351284027, 0.873960554599762, 0.8549960851669312, 0.8459774255752563, 0.05916893109679222, 2.0573253631591797, -0.748338520526886, 0.49546298384666443, 0.17384491860866547, 0.4748014211654663, 0.867651104927063, -0.31527218222618103, 0.11415354907512665, 0.15126238763332367, -0.4690072238445282, -0.6508928537368774, -0.3674915134906769, -0.892035722732544, 0.0073199085891246796, -1.512371301651001, -1.583984136581421, -0.5457629561424255, -0.5305339097976685, 0.048204440623521805, -2.0619142055511475, -0.9815773963928223, 0.8372617363929749, -0.23661407828330994, 1.0709885358810425, -0.1625741571187973, -1.126128911972046, -0.5250599384307861, 0.2363043874502182, 1.4824047088623047, 0.42895185947418213, 2.324723482131958, -0.1477147489786148, 2.7559397220611572, 1.592687726020813, -0.36722174286842346, 1.201673984527588], [-2.151909351348877, 0.01551717147231102, -1.2779380083084106, -0.8656406998634338, -0.7068541646003723, 0.5440430641174316, 0.5693144202232361, 1.0410847663879395, 1.1744030714035034, 0.7083932757377625, -0.20851892232894897, 0.016647327691316605, 0.6526951193809509, -0.42985114455223083, 0.41357529163360596, -0.22194236516952515, 0.5273691415786743, -1.5117101669311523, -0.12849386036396027, -0.24779994785785675, -1.2792282104492188, 0.6251918077468872, 1.2904438972473145, 1.5318140983581543, 2.1858484745025635, 0.5141278505325317, -0.9540798664093018, 1.1603351831436157, 0.1466986984014511, 1.4219095706939697, 0.879364013671875, 1.5714876651763916, -0.0641849935054779, 0.12427960336208344, -1.3517482280731201, 0.6384266018867493, -1.0776394605636597, -0.18135683238506317, -2.278780698776245, 0.39391928911209106, 0.8694117665290833, 1.6541008949279785, 0.05674755573272705, 0.8586269021034241, -0.4814856946468353, 0.3955405652523041, 0.7248467206954956, -1.2850269079208374, -0.43045341968536377, 0.7749866843223572, -0.6682711839675903, -1.0397915840148926, 0.9430465698242188, -1.7604247331619263, -1.1822209358215332, -0.188938170671463, -0.2945486605167389, -0.46884357929229736, -0.9145142436027527, 0.04948122054338455, -1.2255096435546875, -0.49663370847702026, 2.3714325428009033, -1.6205288171768188], [0.9857668876647949, 0.12330368161201477, -0.5031965970993042, 0.5311620235443115, 0.5415318012237549, 0.0831807479262352, 0.4681818187236786, 0.18254558742046356, -0.15777142345905304, -0.29445284605026245, 0.39655497670173645, 1.6613634824752808, -0.6346033811569214, -1.109932541847229, 0.8820720911026001, 0.4953044354915619, 0.8555769920349121, -0.103215791285038, 2.088428497314453, -0.03901268541812897, 0.0451289564371109, -1.2572143077850342, 0.43762972950935364, 0.8720124959945679, 1.382407784461975, -1.2201873064041138, 0.7517261505126953, -1.1578271389007568, -0.3306907117366791, -0.8824644088745117, -0.23698881268501282, -1.5898494720458984, -0.7062409520149231, -1.5127861499786377, -1.4686059951782227, -0.9251951575279236, 0.3163812458515167, 0.36534374952316284, 0.5576224327087402, -0.4233911335468292, 0.7490829229354858, -1.104095458984375, 0.3956204354763031, 0.6044902801513672, 0.9130794405937195, 0.6836671233177185, 1.2886539697647095, 1.2238608598709106, 0.6506466865539551, -0.15203352272510529, -0.8014387488365173, 1.0983376502990723, -0.20172321796417236, -0.41258320212364197, 0.4805472791194916, 1.1102012395858765, -1.1238631010055542, 0.4638976454734802, 1.5352952480316162, -0.256836473941803, 1.0569556951522827, 0.6696733832359314, -2.1199889183044434, 0.2840375006198883], [0.5720909237861633, -0.4899042844772339, -0.5140661001205444, 1.8023054599761963, 1.2873860597610474, 0.2525503933429718, -0.7743906378746033, 0.35759437084198, 0.06677164137363434, -0.9196540117263794, 0.18194551765918732, -0.31161099672317505, -0.07174603641033173, -0.5238351821899414, 0.9156054854393005, 1.4400684833526611, -0.09765686094760895, -0.21479427814483643, -1.0337536334991455, -0.10887453705072403, -1.2933675050735474, 0.1067335307598114, -0.48096951842308044, 0.1292465478181839, -0.45834994316101074, 0.3981558680534363, -0.03435659408569336, -1.366382122039795, 1.8083560466766357, -0.5378581881523132, -1.2227660417556763, 1.3722444772720337, -0.6342455744743347, -0.5626053810119629, 0.1413474828004837, 1.3773703575134277, -0.38029733300209045, 1.1732760667800903, -0.06962895393371582, -0.5120643973350525, 0.07421796768903732, -0.43447813391685486, 1.3332430124282837, -1.4684206247329712, -0.9690811634063721, -0.730807363986969, -0.8231068849563599, -0.276175320148468, -0.7809324264526367, 0.21716293692588806, 0.9849748015403748, 0.383571594953537, 1.2054483890533447, -0.876710832118988, -1.4250342845916748, 0.6102606654167175, 1.8153882026672363, 0.06622438877820969, -0.2520865499973297, 1.7864384651184082, 1.3872973918914795, 1.3062382936477661, -0.8840542435646057, 0.11274838447570801], [0.8877884149551392, 0.13668832182884216, 0.9421136379241943, -0.19994056224822998, -0.01238964218646288, -1.4122499227523804, 0.7242147326469421, -0.39806845784187317, -0.5221479535102844, 0.5367841124534607, -1.0172877311706543, 0.8057448863983154, 0.9710453152656555, -0.6424762606620789, -0.7416545152664185, 0.030571164563298225, -0.6153233051300049, -0.5370203852653503, -1.7546523809432983, -0.45019081234931946, -1.2732652425765991, 0.21295993030071259, -0.05957922711968422, 1.5344599485397339, 0.7083619832992554, -0.6663709282875061, -0.041572801768779755, 0.361100435256958, 0.4343782067298889, -0.34302178025245667, -0.19385093450546265, 0.9627683162689209, -1.1223599910736084, -0.34941038489341736, 1.5625373125076294, 1.0994144678115845, -2.0720112323760986, -0.5960181951522827, -1.1086158752441406, 0.037649061530828476, 0.15178550779819489, 0.2983713746070862, -0.12182961404323578, 0.7350504994392395, 1.5526398420333862, -1.593743920326233, -0.900098979473114, -1.5572770833969116, -1.0226333141326904, 0.2574984133243561, 1.4888672828674316, -1.4147042036056519, 0.9182530045509338, 1.7677441835403442, 0.3380424380302429, -0.6507638692855835, 0.9752421975135803, 0.19292762875556946, 0.4500740170478821, 1.7376751899719238, -1.0377025604248047, 1.2946186065673828, 0.7306241989135742, 0.8460853695869446], [-0.578927218914032, -0.13539686799049377, 0.3437611758708954, -0.40323886275291443, -0.6163045763969421, -0.09271115809679031, -2.691039800643921, -1.1761770248413086, -0.7010757327079773, -0.6478078961372375, 1.04152512550354, 0.24581648409366608, -0.5617290735244751, -1.474036693572998, -1.446663737297058, -1.27853524684906, -0.6641950607299805, 0.023447003215551376, 1.2516478300094604, 1.092345118522644, -0.39148399233818054, 0.3649143576622009, -0.33362460136413574, 0.139659121632576, 0.481925368309021, 0.6439937353134155, 0.4235137104988098, 1.5694375038146973, 1.1704941987991333, -1.7920366525650024, -0.41295865178108215, -0.7192502617835999, 0.7096297740936279, -0.874178946018219, -1.0291967391967773, 0.5100562572479248, -2.5541396141052246, -0.6079098582267761, 0.7151321768760681, 0.4387178421020508, -1.7989100217819214, 0.5855785012245178, -1.180893063545227, -0.5006892085075378, -0.8028594851493835, 1.5835832357406616, 0.16638080775737762, -0.40059977769851685, 0.20442146062850952, 0.13789871335029602, -0.27278220653533936, 0.9981268048286438, -0.898318886756897, 0.2892517149448395, -0.19801077246665955, 0.18654561042785645, 0.8457605242729187, 1.3720906972885132, -0.6046403646469116, -0.4167726933956146, -0.9465067386627197, -1.9327964782714844, 2.200571298599243, 0.7371566891670227], [-2.083524465560913, -0.9903407096862793, 0.657471239566803, 1.2136956453323364, -0.731645405292511, 0.7213078737258911, -1.6726301908493042, -0.10778779536485672, -0.4159485399723053, -0.9478669762611389, -2.010185718536377, -0.7060818076133728, -1.1526358127593994, -0.7990084886550903, 2.9690792560577393, 0.1754421889781952, -0.41509583592414856, 1.4999682903289795, 1.0343921184539795, 0.28785309195518494, 0.4504587948322296, 1.958514928817749, -1.0732219219207764, -0.9384247660636902, -0.5449578166007996, 1.3759229183197021, -0.8702929615974426, -0.6893106698989868, -1.133590579032898, -2.3677592277526855, -1.42583167552948, -0.4636048376560211, 0.13790318369865417, -1.5069758892059326, 1.0644358396530151, 1.1610840559005737, 0.6374610662460327, 1.0913251638412476, -0.2524682581424713, -0.25253432989120483, -0.20655441284179688, 1.0374506711959839, 0.332121878862381, 1.2607712745666504, 0.8573312759399414, 1.9123435020446777, -1.0890247821807861, 0.27809059619903564, 0.370131254196167, -0.2551259398460388, 1.7338550090789795, 0.0015613202704116702, 0.807296097278595, 0.7553043961524963, 0.36728382110595703, -1.6800004243850708, 0.6289582848548889, -0.4008908271789551, 0.8738171458244324, -1.0463374853134155, 1.2689845561981201, 0.26576343178749084, -0.9227412343025208, -1.605378270149231], [1.644256591796875, -1.78407621383667, 0.20754554867744446, 0.6447615027427673, 0.24288149178028107, -0.6905638575553894, 0.846184492111206, 0.8259817361831665, -0.6783069968223572, 0.73187255859375, -1.4043607711791992, -0.05984244868159294, 1.399353265762329, 0.06902909278869629, -0.4226248860359192, -0.60931396484375, 1.4488630294799805, -0.08418840169906616, 0.27990251779556274, -0.8847619891166687, -0.7505688071250916, 0.30351853370666504, -2.4029369354248047, -0.33478739857673645, 0.6820784211158752, -0.9088311195373535, 1.4960969686508179, 0.8730770945549011, 0.16617295145988464, 0.7253274917602539, -0.888817548751831, 1.6178326606750488, 1.7134705781936646, -1.2638744115829468, -0.2889918386936188, 1.2102779150009155, -0.0148318475112319, -0.2998064458370209, 0.5736985802650452, -0.9328187108039856, -0.35237064957618713, 0.7661028504371643, -1.2489842176437378, 0.859058141708374, -0.1959162950515747, 1.4480067491531372, -1.2588412761688232, -1.915115237236023, -0.12558777630329132, 0.7424808144569397, 0.22604061663150787, 0.3562653064727783, 0.6656924486160278, 1.7904187440872192, 0.7133404612541199, 0.5747907161712646, -1.133607268333435, -1.8824692964553833, -1.0760127305984497, -0.4977339506149292, -0.8378555178642273, 1.04770827293396, 0.2673450708389282, 0.8276536464691162], [-1.3827263116836548, 0.8057337999343872, 0.2581728994846344, -0.4621685743331909, 0.45866385102272034, 0.2672533690929413, 1.0877351760864258, 0.37413808703422546, -1.4168809652328491, -0.28353261947631836, -0.17247913777828217, -0.08487199991941452, 0.35422590374946594, -1.1010420322418213, -1.021506667137146, -0.5537379384040833, -0.703863263130188, 0.8845812082290649, -0.20730580389499664, 0.8643648028373718, -0.25991737842559814, 0.7423173189163208, -0.04397124797105789, -0.8578015565872192, -1.0880465507507324, -0.04163702577352524, 1.6908735036849976, -0.1557493507862091, -0.42069560289382935, -1.6278650760650635, 0.6149244904518127, 0.34580910205841064, -1.694707989692688, 1.365354299545288, 0.5251456499099731, 0.9675611257553101, -0.17169976234436035, 0.12611399590969086, -0.3721948266029358, 1.5422903299331665, -0.32838428020477295, 0.467267245054245, 0.8082560300827026, -1.3537052869796753, 2.1906237602233887, -0.8953049182891846, 0.3595046103000641, 1.1019333600997925, 1.4775536060333252, -1.8250682353973389, -0.5238676071166992, 1.620073676109314, -0.47906917333602905, 0.07655983418226242, -0.06388608366250992, 1.3138478994369507, -0.9327305555343628, -0.7307640910148621, 0.4989672601222992, -0.0038535657804459333, -1.429104208946228, 0.8430569767951965, -0.5022851824760437, -0.36646175384521484], [-0.9549401998519897, 0.6149780750274658, -0.3893008232116699, -0.43155157566070557, -0.6674374938011169, 1.3852227926254272, -1.0992710590362549, 1.3599889278411865, -0.17995581030845642, 0.17850598692893982, -0.8654782176017761, 0.002813511760905385, 0.01746441423892975, -0.011491650715470314, -1.351400375366211, 0.12678536772727966, -1.400456190109253, 1.338422179222107, 0.48845311999320984, 0.21701931953430176, 0.9042980670928955, -0.3399394750595093, -0.5193068981170654, 0.7634779810905457, -0.33324694633483887, 1.6458982229232788, 0.0258611049503088, 0.9355031251907349, 0.7687302827835083, -0.6128727793693542, -0.48494961857795715, 1.0705227851867676, -0.13455601036548615, -0.0933089330792427, -0.44128355383872986, 0.578326404094696, 1.2443293333053589, -1.3764065504074097, -0.4546332061290741, -0.7787817120552063, 0.6937719583511353, 1.019477128982544, 0.496611088514328, 0.340623676776886, -0.9260802268981934, -1.016682744026184, 0.5683107972145081, -0.8914915323257446, 0.14954839646816254, -0.7049671411514282, -0.574394166469574, 0.4102502763271332, 0.7781442999839783, -0.48141682147979736, 1.2408957481384277, -2.3968822956085205, -0.8400547504425049, -0.2696072459220886, 1.0777721405029297, 0.07079049944877625, 0.8306252956390381, -0.4995407462120056, 1.3372843265533447, -0.031495507806539536], [0.6225089430809021, -0.8917463421821594, -0.4310557246208191, -0.21589581668376923, 1.6596230268478394, -0.1059584990143776, -0.5019925236701965, -1.358066201210022, -0.009161793626844883, -0.44871318340301514, 0.6654516458511353, 0.19718872010707855, 0.9908804297447205, 1.3645200729370117, -1.66499662399292, 0.5359081029891968, 0.21835866570472717, 0.23855140805244446, 1.0100815296173096, -2.3798391819000244, 0.9898349046707153, -0.18042877316474915, 0.7702791094779968, 0.40418869256973267, 0.45738592743873596, 0.8650065660476685, -0.20028162002563477, -0.22879694402217865, 0.5237828493118286, -0.7779150009155273, 0.8036746382713318, -0.8889484405517578, 0.11793556809425354, 1.6885888576507568, 0.7270112633705139, -1.3138937950134277, 0.08228973299264908, 1.0060416460037231, 0.2956957221031189, -1.0417159795761108, -0.5863679051399231, -1.0602463483810425, -0.43986019492149353, 0.6786286234855652, -0.3397332727909088, -0.7025966048240662, -2.0060389041900635, -0.1192978098988533, 0.18821895122528076, -0.1428719013929367, -0.4869031608104706, 0.6481332778930664, 0.7375412583351135, 0.3383292853832245, 0.4875209331512451, -0.6397995352745056, 1.0277608633041382, 0.9111039638519287, -0.39328914880752563, -0.5259580016136169, -0.5587117671966553, 0.811357855796814, -0.26557233929634094, -0.44361409544944763], [1.5638012886047363, -0.3220078647136688, 0.6909499764442444, -0.5361567735671997, 1.3559409379959106, 0.22787104547023773, -2.24231219291687, 0.18290357291698456, -0.31875330209732056, -1.1120327711105347, 0.5297223329544067, 1.1721397638320923, -0.27124151587486267, -1.5179312229156494, 2.6055073738098145, 0.5299402475357056, 0.8194693326950073, -0.07617561519145966, 3.2913880348205566, 0.8003538846969604, -0.43334823846817017, 0.44338756799697876, -0.6369744539260864, -1.5292555093765259, -0.3376024663448334, 0.7411221265792847, -0.196807399392128, 0.08500656485557556, 3.0693905353546143, 0.13100431859493256, 0.7707178592681885, -0.039449293166399, 0.1099071055650711, 0.7701690196990967, 0.9630838632583618, -1.0147833824157715, 0.42043086886405945, -0.7878987789154053, 0.826465904712677, -0.00996309146285057, 1.3274656534194946, 1.4577046632766724, 0.0010958312777802348, -0.33635419607162476, 0.5473840236663818, 1.4263566732406616, 2.046614408493042, 0.9779314398765564, -0.061080340296030045, 0.52345871925354, 1.4174327850341797, -1.8621711730957031, -1.047236680984497, 2.5030198097229004, 0.5480148196220398, -1.371826171875, -0.21831178665161133, 1.472079873085022, 0.3329843580722809, -2.0459957122802734, 0.9633486866950989, 1.5954694747924805, 1.2378358840942383, 1.0942600965499878], [1.6264077425003052, 0.15516890585422516, 0.45063847303390503, 0.7063334584236145, 0.034517545253038406, 1.789541244506836, -0.07498877495527267, -0.6804020404815674, -0.11043678969144821, 0.916438102722168, 1.2424105405807495, 0.2576638162136078, 0.32101911306381226, 0.8028551340103149, -0.0888328105211258, 0.4987819194793701, -0.1904015690088272, 0.9700416922569275, -0.1884269416332245, 1.2852438688278198, -0.8068927526473999, -0.4482598900794983, 0.7359937429428101, 0.1856009066104889, -0.1321808248758316, 1.3689332008361816, 2.7043356895446777, 0.4151212275028229, -0.683295488357544, 0.029668301343917847, 1.321748971939087, 0.15866678953170776, -0.32037585973739624, -0.9161102175712585, 1.6023046970367432, -0.14040511846542358, 0.11135151982307434, 0.4193763732910156, 0.6520407199859619, 0.7525503635406494, 1.7558106184005737, 0.04779139533638954, -0.07321300357580185, 1.2497992515563965, -1.122794270515442, -0.9979522228240967, 0.3781026303768158, -0.3735138177871704, 0.718788743019104, 1.185333013534546, -0.41532355546951294, -0.13423727452754974, -0.6598539352416992, 0.8471766114234924, -1.4497699737548828, -0.9016344547271729, 0.8544948101043701, -0.8206877112388611, 1.0139967203140259, 0.7001307010650635, 0.6626254320144653, -0.3941114544868469, -0.4773959815502167, 0.41730889678001404], [0.29045236110687256, 0.5648941993713379, -0.160626620054245, -0.050696562975645065, 0.47446075081825256, -0.9079801440238953, 1.314400315284729, 0.746342658996582, 2.117709159851074, 0.5396052598953247, 0.20986731350421906, -0.7613154053688049, -0.52772456407547, 0.3818973898887634, -0.6693539023399353, -1.7530279159545898, 0.5620514750480652, 1.2139309644699097, 1.2134861946105957, -0.5670505166053772, -0.5816474556922913, 0.20574840903282166, -1.163684368133545, -0.3968208432197571, 0.06413709372282028, 0.254107803106308, 0.5528077483177185, -1.7558457851409912, 1.4073964357376099, 1.3187581300735474, -0.21667686104774475, 0.34801313281059265, -0.9430084228515625, 0.5351950526237488, 0.0727267935872078, 1.8183417320251465, 1.3199529647827148, 1.1985291242599487, 0.27247267961502075, 0.5655320882797241, 2.2435362339019775, 0.3326556384563446, -1.560786247253418, 0.4464927315711975, 0.4435127377510071, -0.5572549104690552, -0.4693205654621124, -0.30172061920166016, -0.6342882513999939, 0.6395822167396545, -1.124807596206665, -0.8531618118286133, 1.2071336507797241, 1.081618070602417, 0.8193563222885132, -0.1784590631723404, -1.1799718141555786, 0.12630905210971832, 0.12711496651172638, -0.8508651256561279, 1.6206409931182861, 0.7970332503318787, -0.018117131665349007, -0.41208383440971375], [-0.8003345727920532, 3.0739517211914062, -0.2773374319076538, 1.3431813716888428, -0.42467623949050903, -0.8726112842559814, 0.3495660424232483, 1.8333816528320312, 0.22100992500782013, 0.9530061483383179, -0.9911995530128479, 0.3194952607154846, 0.2651768922805786, 1.2393858432769775, -1.1902666091918945, 0.5085002779960632, 1.1220664978027344, -0.4418807029724121, 0.2192913293838501, -1.0006206035614014, 0.838021457195282, 0.7025697231292725, 0.42063483595848083, -0.7379311323165894, -0.7088823318481445, 0.768134355545044, 1.2165824174880981, 0.09439793229103088, -1.104880928993225, 1.4170726537704468, -0.15848419070243835, 2.008776903152466, 0.3333250880241394, 0.14365145564079285, 1.7167409658432007, -0.8246092796325684, -0.49934449791908264, -0.6858195066452026, 1.9026470184326172, 0.9059478044509888, -1.5421127080917358, -0.48235252499580383, 0.4765941798686981, 0.6054137945175171, -0.45239830017089844, 0.8545050621032715, -1.9963257312774658, 0.9185309410095215, -0.34090456366539, 0.01824544370174408, -1.0198390483856201, 0.7949976921081543, -1.9637038707733154, 0.6716402173042297, -1.7858259677886963, -1.5068247318267822, -0.7602339386940002, -1.9065710306167603, -1.4554146528244019, -0.6439120769500732, -0.36046603322029114, -1.0275901556015015, 0.3090899884700775, -0.11356639862060547], [1.2525615692138672, -1.2061183452606201, 0.729995608329773, -0.7321957349777222, 0.8304445147514343, -0.293876588344574, 0.5637465119361877, -0.21946009993553162, 1.448895812034607, -0.3414573073387146, 0.523052990436554, -0.026957262307405472, -1.036847472190857, 0.9575844407081604, -1.2207306623458862, -0.7793827652931213, 0.7725741863250732, 1.335002064704895, -1.0764260292053223, 0.8846490979194641, 0.019067851826548576, 0.40550804138183594, 0.6026955842971802, -0.4033856987953186, -1.1816458702087402, 0.7286495566368103, 1.1371008157730103, -0.8298709988594055, -1.168777346611023, -1.9078736305236816, 1.0424998998641968, -0.6805914044380188, -0.3548152446746826, 0.444423645734787, -2.0331897735595703, 1.497496485710144, 2.2191390991210938, 1.087995171546936, -1.6188400983810425, 0.460734099149704, -0.4256461560726166, 0.9375272393226624, -1.5664474964141846, 0.14947184920310974, -0.6819789409637451, -0.383980393409729, -0.16149114072322845, -0.44110366702079773, -1.1484062671661377, -1.326528549194336, -0.23517051339149475, -0.6549341082572937, 0.2532193660736084, -0.66783607006073, -0.005111214704811573, -0.6052586436271667, 1.6229000091552734, 1.2401247024536133, 0.6759928464889526, 0.11588604748249054, 0.05713644251227379, -0.09946833550930023, 0.7532430291175842, 1.3423991203308105], [0.9793369174003601, -1.0761964321136475, 0.12972955405712128, 1.093435287475586, -0.3971034288406372, 0.13641121983528137, 1.3863978385925293, -0.097066231071949, -0.8994467854499817, -0.7596908211708069, -0.08469713479280472, 1.0474727153778076, -0.5459469556808472, -0.32528746128082275, 0.4959529638290405, 0.8673844933509827, -0.6567602753639221, -0.4493172764778137, 0.07829725742340088, -0.42420583963394165, -1.963645339012146, 0.6332525610923767, 3.187134027481079, 0.7914826273918152, 0.9980902075767517, 0.22002655267715454, -1.0425260066986084, 0.16979847848415375, -1.5616101026535034, -0.4775759279727936, -0.30301806330680847, 1.2443194389343262, 0.04205423966050148, -0.4361800253391266, -0.5237106084823608, 0.11342599987983704, 0.15255950391292572, 0.3418119251728058, 0.2016705870628357, -1.1191924810409546, 0.13764183223247528, 0.46184006333351135, 1.6509422063827515, 1.9100468158721924, -2.0479490756988525, -1.195798635482788, 1.1486413478851318, -0.5016000866889954, 2.1687207221984863, 0.43575459718704224, 1.0214189291000366, -0.4975423216819763, 1.5944305658340454, -0.48628950119018555, 0.06712799519300461, 1.256783127784729, -2.270684003829956, 0.5073159337043762, -0.21824447810649872, -0.46091705560684204, 0.7097218036651611, -0.36088982224464417, -0.37592071294784546, -1.4032822847366333], [1.1190420389175415, 2.546703577041626, 0.7107159495353699, -1.0429010391235352, -2.3637564182281494, -0.11252455413341522, 0.3907144069671631, -0.22160185873508453, -2.416947841644287, 1.2961657047271729, -0.20822662115097046, -1.0638391971588135, -0.1047806367278099, -0.2723923623561859, -1.4277474880218506, 0.5098581314086914, 1.4753161668777466, -0.34451887011528015, -0.6478651165962219, -0.8236644864082336, -1.8023065328598022, 0.9357530474662781, 0.5359717607498169, -2.194385528564453, -0.7170450091362, 0.02957775443792343, -0.016924748197197914, -1.1310497522354126, -0.039104435592889786, 0.9556571245193481, -1.1775732040405273, 0.9457593560218811, 0.3767072260379791, 0.27973273396492004, 1.173090934753418, 0.21680594980716705, 0.612924337387085, -0.5144392251968384, 1.000442624092102, 0.3554537296295166, -1.3398622274398804, 0.02462380938231945, 0.5585426092147827, -0.585852861404419, 0.3282444477081299, 0.7763580083847046, -0.39893656969070435, 0.24767622351646423, -0.7062408924102783, 1.3333126306533813, 0.1335754543542862, -0.7687544226646423, 0.3145882189273834, 0.2992130219936371, 0.8273759484291077, -1.6095637083053589, 1.6172738075256348, -2.5624630451202393, -3.3696541786193848, -0.1550740748643875, -0.8192278742790222, -1.244564175605774, 0.7721660137176514, -0.7805843949317932], [-0.41202718019485474, 0.4763827621936798, -0.5608438849449158, -0.07934913784265518, 0.8222856521606445, -0.9641782641410828, 0.5692228078842163, 1.7592114210128784, 1.221925973892212, -0.2950810492038727, 1.269262671470642, 0.8813818097114563, 1.145391821861267, 0.30167052149772644, 1.3148417472839355, 0.24873019754886627, -1.3926366567611694, -0.24402396380901337, 0.9923849105834961, -0.8062727451324463, 1.0904353857040405, -2.095533609390259, 0.9322154521942139, 0.11991298198699951, -0.8216806054115295, 2.591998815536499, 0.351189523935318, 0.8668246865272522, 1.1970857381820679, 2.173311471939087, -1.3692922592163086, 0.8709970116615295, 0.5023108720779419, 2.570319890975952, 0.1746366173028946, -0.3027009963989258, -0.8378618955612183, 0.46095478534698486, 1.5161939859390259, 1.0091487169265747, 0.9596680998802185, 1.1320477724075317, 0.03992069512605667, -0.39034461975097656, -1.5131765604019165, 1.2536205053329468, -0.10745308548212051, -2.5947821140289307, -0.34055227041244507, -1.1778641939163208, 1.680064082145691, -0.6790658235549927, 1.4881699085235596, 1.517349123954773, -0.5588741302490234, -0.5446581244468689, 0.0035645603202283382, 0.03173185884952545, -0.2934824824333191, -0.5138590931892395, -0.4111778140068054, -2.62323260307312, -1.2043999433517456, -0.34945130348205566], [-0.06625493615865707, -0.25479891896247864, 0.7392596006393433, 0.5343311429023743, -0.3682486414909363, 0.3782489597797394, 2.2021799087524414, 0.9496263861656189, -0.42935654520988464, -0.4690735936164856, 1.1871893405914307, -0.6773525476455688, -0.9369813203811646, -0.12696650624275208, -1.3911124467849731, -0.7526956796646118, 0.726894199848175, -1.0502958297729492, 0.01231094915419817, -2.7026329040527344, 0.028325606137514114, 0.16838620603084564, -0.404528945684433, -1.1721827983856201, 0.8572167754173279, 0.2793921232223511, 0.8465414643287659, 1.0069208145141602, 0.19164353609085083, 0.7543870806694031, 2.0011043548583984, 0.11412813514471054, -0.12264163792133331, 1.132449984550476, 0.6794425845146179, -0.9593403935432434, 0.6470091342926025, -1.8747459650039673, 1.3294570446014404, -0.34915831685066223, -0.35184505581855774, 0.4518928527832031, -1.073730230331421, -0.36243388056755066, 0.34425410628318787, 1.260859727859497, 1.633083462715149, -0.713724672794342, 0.3309512138366699, -1.8981378078460693, -1.164426326751709, -0.5127401351928711, -0.11271266639232635, -0.4711674451828003, 0.27505016326904297, 1.1035454273223877, -0.43937423825263977, 1.955058217048645, -0.3559858500957489, 1.3225460052490234, -2.7071192264556885, -0.43763554096221924, -0.6608431935310364, 0.11732862889766693], [-2.0539662837982178, -0.9110720753669739, 0.6534852981567383, -0.6776129007339478, 0.6235578656196594, -1.8949329853057861, 0.07177357375621796, -0.047637782990932465, 1.838435173034668, -1.104435682296753, -0.49731484055519104, 0.6664541959762573, 0.5168073177337646, -1.019959568977356, -0.7915239334106445, -0.2790504992008209, 0.26037073135375977, -1.5032447576522827, 0.41494956612586975, 2.46820330619812, 0.4929973781108856, -0.8018167614936829, 0.4607159495353699, -0.2041337490081787, -0.794402003288269, 0.31149765849113464, 0.47021088004112244, -1.388615369796753, -0.27270883321762085, -1.192659616470337, -0.07576961815357208, 1.1984856128692627, 0.7085554599761963, 1.169356346130371, -0.028775986284017563, -0.45034515857696533, -0.17331339418888092, 1.0577508211135864, 0.45497027039527893, 0.8161460161209106, -1.1737347841262817, 0.3800402879714966, -0.38343510031700134, -0.8852211236953735, -0.543491542339325, 1.3784693479537964, -0.48031967878341675, -0.3038738965988159, -0.3486631214618683, -0.6430991888046265, 0.03797898441553116, 0.6702562570571899, -2.2939932346343994, 0.9978323578834534, -0.15980996191501617, 0.5750547647476196, 0.38198795914649963, -1.2413678169250488, -0.19993674755096436, 0.08450271934270859, -1.0645440816879272, 0.5235722064971924, 0.0789477676153183, -0.42727941274642944], [0.5497738718986511, 0.597727358341217, -0.11962353438138962, -0.40247389674186707, -0.4036649167537689, -0.16662180423736572, 0.7885437607765198, -0.8682759404182434, -1.0326604843139648, -1.6011775732040405, -0.14736595749855042, 0.01437135599553585, 0.19090090692043304, -0.07671947777271271, -1.964412808418274, -1.2920260429382324, -0.7362580895423889, 0.32440370321273804, 0.2933613359928131, -0.1351093351840973, -1.0145182609558105, 1.4031044244766235, -0.8271932005882263, -0.7264096140861511, 2.3493478298187256, 0.03156643360853195, -0.40677696466445923, 1.6831932067871094, -0.23682095110416412, -0.10290583968162537, 1.7070388793945312, -1.1411341428756714, -0.37092217803001404, -0.3627868592739105, 2.6209065914154053, 0.26607730984687805, 0.8009966611862183, 0.760715901851654, -0.3131648600101471, -1.5722851753234863, -0.683445394039154, -1.4567465782165527, 2.029254198074341, -0.3396996557712555, 1.2667250633239746, 0.5609294176101685, -0.2552785873413086, -0.36509615182876587, 1.5602805614471436, -1.0001407861709595, 0.19062553346157074, 0.9362952709197998, -1.8866922855377197, 0.4077633321285248, -0.7316771745681763, -0.5061376690864563, -0.09765007346868515, 0.7425392270088196, 0.6028452515602112, -0.5793525576591492, 0.763126015663147, -2.6208019256591797, -2.5009727478027344, 0.5365344285964966], [-0.22505123913288116, 0.8236535787582397, -0.7520629167556763, -1.2067596912384033, -0.09578090906143188, -1.4525256156921387, -0.5750902891159058, 0.6273534297943115, -0.016745077446103096, 0.4037765860557556, -1.6300524473190308, -0.7541630864143372, 0.3105982542037964, 0.36063334345817566, -0.009823627769947052, 1.0988825559616089, 0.5945262312889099, 0.1839628666639328, 0.36036086082458496, 0.16248367726802826, -0.42617425322532654, -1.7895442247390747, 0.46652358770370483, 0.7106462121009827, -1.3637621402740479, -1.2445369958877563, -0.29573968052864075, 0.27389097213745117, 0.2255653589963913, -0.14081737399101257, 0.5832534432411194, -0.16974423825740814, -1.9080169200897217, 0.254294753074646, 0.07649052143096924, -1.3328946828842163, -0.11282116919755936, 0.2508951723575592, 0.13058193027973175, 2.06722092628479, -0.7338043451309204, 1.2222392559051514, -0.3078814744949341, -1.198189616203308, 0.9676984548568726, 1.8732656240463257, 0.6172453165054321, 0.5872587561607361, 0.8006206750869751, 1.626202940940857, 1.1871058940887451, 0.5320218801498413, 0.07860229909420013, 1.1469752788543701, 0.009727134369313717, 2.2595746517181396, 0.7395470142364502, -0.39826905727386475, -0.9571064114570618, -1.3211565017700195, -1.6978727579116821, -1.1030174493789673, 1.465325117111206, 0.43776407837867737], [0.49289000034332275, -1.1650748252868652, -0.11792290210723877, -0.4057656526565552, 0.6340444087982178, 0.15941055119037628, -1.0227060317993164, 0.6650516986846924, 0.9183769822120667, 1.2931489944458008, -2.4585280418395996, -1.4821593761444092, -0.3889947235584259, -0.028746040537953377, -0.16238901019096375, -0.18487507104873657, 1.7569993734359741, -2.330071449279785, 1.7482056617736816, 2.2847421169281006, -1.2628031969070435, 0.8364963531494141, -0.47954049706459045, -0.2779557704925537, -0.9598592519760132, -0.2723383903503418, 0.006491669919341803, -1.125374674797058, -1.3104729652404785, -0.7489739060401917, 0.2866577208042145, 2.0165023803710938, -0.5528672337532043, -0.78031986951828, 0.7110437154769897, 0.08436915278434753, -1.4445182085037231, 1.1016106605529785, -0.5034974217414856, 1.2328319549560547, -0.06438937783241272, -0.4544468820095062, -1.256913661956787, 2.361790180206299, 0.6220779418945312, 2.2185323238372803, -2.1777403354644775, -0.7302411198616028, 0.13510462641716003, 0.8801645636558533, 0.6474248766899109, -0.5450482964515686, 2.058839797973633, -0.10983595252037048, 0.3850654661655426, -0.8117324113845825, 1.0826984643936157, 0.8435529470443726, -0.2066870629787445, -0.6787363290786743, -1.2250066995620728, 0.4429556727409363, -0.145328551530838, -0.4615073502063751], [0.3480805456638336, 2.1828107833862305, -0.20901204645633698, 0.052151333540678024, -0.038955919444561005, -1.409756064414978, 0.1392878144979477, 0.4863525927066803, -0.1556505560874939, 0.9456501603126526, 0.9060636162757874, 1.211693525314331, 0.7273733019828796, -1.8428826332092285, 1.574580430984497, -0.31486162543296814, -0.35480666160583496, -1.5500037670135498, 0.9135758280754089, -1.3842650651931763, 2.194063663482666, 0.304431676864624, -0.2325488030910492, 2.0302252769470215, 1.222826600074768, -0.3430600166320801, 1.440202236175537, 1.3145718574523926, -1.1528549194335938, 0.567611813545227, 0.8030272722244263, -0.42639535665512085, -0.23233246803283691, -1.1528615951538086, 1.2143441438674927, -1.1923840045928955, 1.196898341178894, -0.14272618293762207, 0.11563178151845932, -0.6288104057312012, 2.191969394683838, -0.07744099944829941, -0.41411545872688293, -0.14863410592079163, -1.5597155094146729, 0.7989278435707092, -2.308455467224121, -0.6755734086036682, -0.7695436477661133, 0.8455827236175537, 1.0675569772720337, 2.1623921394348145, 1.392468810081482, 0.49534764885902405, -0.7604480981826782, 0.548976719379425, 2.567845344543457, -0.5181111693382263, -0.6837804913520813, -1.5882713794708252, 0.37864506244659424, 1.27643620967865, -0.9428830146789551, 1.6345090866088867], [0.2950354516506195, -0.6008802056312561, -0.29327788949012756, -0.06940191239118576, -1.507582187652588, 1.5929524898529053, -0.09144474565982819, 0.29142892360687256, -0.6918917894363403, -0.5735728740692139, -0.9232064485549927, 0.7144014239311218, 0.051533304154872894, 0.7401901483535767, 2.56978440284729, -0.19396743178367615, 2.056569814682007, 1.478463888168335, 1.32172691822052, 0.37935158610343933, -0.5855051279067993, -1.053026556968689, 0.542953372001648, -0.4512978494167328, -1.443160891532898, 1.0273840427398682, 0.05133720859885216, -0.11877810209989548, 1.4746904373168945, -0.34056779742240906, 0.5903337001800537, 0.4747520685195923, -0.050466395914554596, 0.8588889241218567, -1.9207690954208374, -0.4550597667694092, -0.3772772252559662, -0.9123722910881042, -1.8703724145889282, 0.2430192530155182, 1.2036879062652588, -1.2914286851882935, -0.5735601782798767, 0.7460415959358215, 0.19409644603729248, 0.445837140083313, -2.3830230236053467, -0.05509592592716217, -0.029688524082303047, -0.8561134934425354, -1.247995138168335, 0.2832261621952057, 2.55660080909729, -0.9438773989677429, -0.04911842942237854, 0.07464973628520966, -0.2605554759502411, -0.009621890261769295, -0.39366453886032104, -0.31562456488609314, 0.6504176259040833, -0.5566336512565613, -0.9553090929985046, 1.4751402139663696], [-0.20605242252349854, -0.04005051404237747, -0.5637929439544678, 0.5927843451499939, 1.4258275032043457, -1.8348692655563354, 1.0609662532806396, -0.8538082242012024, 0.08571886271238327, -0.10213949531316757, 0.4115986227989197, 1.244879961013794, -1.0631433725357056, 0.8765150308609009, -0.819359302520752, -0.8324199318885803, 0.2275431752204895, -0.20006178319454193, 0.2481233924627304, -0.551308810710907, -0.008618338964879513, 0.29494622349739075, 0.15150929987430573, -1.7009077072143555, -0.3791648745536804, -0.7732191681861877, -0.624734103679657, -1.268512487411499, -0.2690534293651581, -1.1393444538116455, 0.04952172935009003, -0.6421937346458435, -0.10954599827528, -0.3540009558200836, 0.4905015230178833, -1.6871639490127563, 0.008856282569468021, 0.6204884648323059, 0.27881988883018494, 0.29744037985801697, 0.6407867074012756, 1.6991264820098877, -1.8841826915740967, -0.06569533795118332, 0.5414841771125793, -0.18950390815734863, -0.2890245020389557, 0.340839684009552, -0.18341873586177826, -0.562468945980072, -0.8159339427947998, -0.9681617617607117, 1.6880271434783936, -0.573329508304596, -0.2840736508369446, 0.3982771933078766, -0.17426922917366028, -0.032047074288129807, 0.6870754361152649, -0.20270779728889465, 0.2816406786441803, -1.0200798511505127, 0.5542857050895691, -1.280080795288086], [-0.16814759373664856, 0.1965874433517456, -1.062275767326355, 0.0980476662516594, 0.7628425359725952, -1.3606759309768677, 0.06286387145519257, 0.877770721912384, 0.46710577607154846, -0.872824490070343, 0.5448265075683594, 0.9509842991828918, -0.48015910387039185, 0.29300299286842346, 1.2903234958648682, -1.5155600309371948, 0.5133731365203857, -0.31393226981163025, -1.4068567752838135, -3.4995508193969727, 0.7377650737762451, -0.9654210209846497, 0.7801045775413513, -0.3942510187625885, 0.8956670761108398, 0.246390700340271, -0.2465120404958725, 0.4421168565750122, -0.09912899136543274, -0.6892821788787842, 0.9467089176177979, -2.0294532775878906, -0.2323998510837555, -1.377321720123291, 1.6151938438415527, 1.028363823890686, -0.1481388360261917, -1.5510280132293701, -0.7846812009811401, -1.953263282775879, 0.5603775382041931, 0.869147002696991, 2.1424715518951416, -0.4872764050960541, 0.9645618796348572, 0.404428631067276, -0.4838765859603882, -0.5738930702209473, -1.5964573621749878, -2.768829107284546, -0.25471559166908264, -0.14287878572940826, 1.068785548210144, 0.6277888417243958, -0.0975581705570221, -1.1492640972137451, 0.009126191027462482, -1.0723844766616821, 1.2027487754821777, -0.7504947185516357, -1.5253260135650635, -1.0719692707061768, -0.9255493879318237, 0.15315276384353638], [-2.0532405376434326, 0.10154420137405396, -1.1563268899917603, -0.35409143567085266, 1.2321174144744873, 0.08549439907073975, -0.035003334283828735, -1.0911791324615479, 0.06828301399946213, -1.1843822002410889, 0.7591248154640198, 0.7592605352401733, -0.19352684915065765, -2.045135498046875, -0.5500879883766174, -0.4193914532661438, 0.16138289868831635, 0.5386321544647217, 1.7816230058670044, -0.6468935608863831, -0.37857893109321594, 0.8519774675369263, -1.3043056726455688, -1.0413012504577637, -0.5005406141281128, 0.9547396302223206, 1.1954745054244995, 0.3446200489997864, -0.08622017502784729, -0.3080635666847229, 0.3218235373497009, -1.708654522895813, 0.18831072747707367, -0.10203441232442856, -0.20159688591957092, 1.4725747108459473, -0.16188931465148926, 2.8421003818511963, 0.36085695028305054, -0.09678659588098526, -0.07604032754898071, -0.6525298953056335, -0.03770207613706589, 0.09087210148572922, 0.46923837065696716, 0.7898412942886353, 0.9412949085235596, -0.6220235228538513, 0.5068049430847168, 0.8054354190826416, 0.20488294959068298, 0.5298857688903809, -1.6939111948013306, 0.02156844548881054, -0.7215161919593811, 0.003518662415444851, -0.35452455282211304, 0.22481004893779755, -1.5482922792434692, 0.13794469833374023, -0.4278585612773895, -0.5462290644645691, -0.3860081136226654, 0.7730686664581299], [-2.152432441711426, -0.022802947089076042, -1.4378949403762817, 0.05897116661071777, -1.0018845796585083, 0.811374306678772, -0.323988676071167, -0.11615920811891556, -1.0317703485488892, 0.12420102208852768, 1.0062288045883179, -0.678961992263794, 2.329627513885498, 0.182785302400589, -2.5592517852783203, 0.7295709252357483, -0.7303610444068909, 1.6659358739852905, 0.5429365634918213, 1.5520247220993042, -0.8749749064445496, 0.16481401026248932, 0.6763835549354553, 1.3850890398025513, -0.9569982886314392, -0.7732764482498169, -0.31969478726387024, -0.3506975471973419, 0.31151053309440613, 0.40744268894195557, 0.39174237847328186, 0.9035201668739319, 1.457196593284607, -0.41231340169906616, 1.4789704084396362, -1.3461815118789673, 0.1471439152956009, 0.3580099046230316, -0.8092142939567566, 1.591199278831482, -0.6170219779014587, -0.0627947598695755, -0.5572090744972229, 0.6695098280906677, 0.5046478509902954, -0.6737927198410034, -0.23438890278339386, 2.8687808513641357, 0.4822145700454712, 0.6977007389068604, -0.5078702569007874, 0.02266325242817402, -1.369300127029419, -1.7750738859176636, 1.077610731124878, -0.9298906922340393, 0.11858676373958588, -1.058284878730774, 1.7449918985366821, -1.0471020936965942, 0.5043119192123413, -0.7564653754234314, 0.45943769812583923, 0.05628399923443794], [-1.0366675853729248, -0.3847046494483948, 0.324493408203125, -0.9732152223587036, -0.42828479409217834, -1.490053653717041, -0.2689040005207062, 0.8175351023674011, -0.5652023553848267, 1.4265793561935425, -0.9307888746261597, -0.42510855197906494, -1.1273231506347656, -2.3443682193756104, -0.3637281656265259, -0.6868511438369751, 0.9181003570556641, 0.21850299835205078, 2.7592976093292236, -0.36276155710220337, -0.9513252973556519, 0.06750507652759552, 1.2522519826889038, 1.0555435419082642, 0.29490405321121216, -0.3971377909183502, 1.018303394317627, 0.008271215483546257, -0.28448134660720825, 0.16008271276950836, 0.5890607833862305, -0.24070827662944794, 1.010159969329834, 0.36438247561454773, 0.0007312206435017288, 0.8399525284767151, 0.4159325063228607, -0.16135472059249878, 1.557224154472351, 0.44956302642822266, 2.140146493911743, -0.49475231766700745, -1.097475290298462, -0.6728222370147705, 0.466369092464447, 0.06490196287631989, -0.8030748963356018, -0.593221127986908, -0.5017610192298889, -0.33298078179359436, 0.327490895986557, -0.08662586659193039, 0.4934631288051605, -0.9442703127861023, 0.6928496360778809, 0.8201981782913208, 0.1259126365184784, 0.02505004033446312, 1.1324690580368042, 0.6063265204429626, 0.379537433385849, 1.0097532272338867, -1.2386887073516846, 0.788832426071167], [0.8785626292228699, 1.266850233078003, 2.127643585205078, 0.3123728930950165, -0.7103789448738098, -0.2339131385087967, 0.5318379402160645, 0.07100632041692734, 0.06724970787763596, 0.08921632915735245, -1.1596226692199707, -0.7873283624649048, 0.9023197293281555, 2.295992374420166, 0.7444953322410583, -1.4802525043487549, -0.3807036578655243, 0.5268284678459167, -0.9734468460083008, 1.2778382301330566, -0.3071921467781067, -2.178192615509033, 0.11284715682268143, 0.3247472941875458, 0.8393868207931519, 0.0767185240983963, -0.2151959240436554, -1.609897494316101, -0.5954874753952026, 0.6438161730766296, 0.11916995793581009, -0.9654342532157898, 0.007853973656892776, -0.3387305438518524, -1.266918420791626, -0.9519868493080139, -1.2185243368148804, -0.24046838283538818, -0.5673398375511169, 1.0549522638320923, 1.0552064180374146, 1.023378849029541, -0.038182277232408524, 0.6616647243499756, 0.6452085375785828, -0.9901250004768372, 2.051809072494507, 0.6926209926605225, 1.304087519645691, -0.5610249042510986, -0.545462429523468, -1.7389397621154785, 0.787074863910675, -0.3833668828010559, -1.044567346572876, -0.10860278457403183, 0.7576969861984253, 0.08045904338359833, 0.9106532335281372, 1.6117035150527954, 0.622321605682373, 1.0547542572021484, 0.67746901512146, -0.7374098896980286], [-0.08514435589313507, 0.37534719705581665, -0.8702115416526794, -0.6833168268203735, 0.38787469267845154, 1.0348918437957764, -0.8179505467414856, 1.0958157777786255, -0.06306757777929306, -0.7615209817886353, 0.06256023049354553, 0.2184954434633255, -1.1563578844070435, -0.943015992641449, 0.5795185565948486, 0.20367075502872467, -1.3562862873077393, -1.4864979982376099, -2.0458481311798096, -0.14351263642311096, 0.5226635932922363, -0.6254938840866089, -0.26090750098228455, 0.1937108188867569, 0.8795711398124695, 1.3243403434753418, 0.5924682021141052, 2.180457353591919, 1.620568037033081, 1.7539739608764648, -1.5023698806762695, -0.039639912545681, 0.1954474151134491, 0.9299131631851196, -0.1801321804523468, -2.2358052730560303, -1.1757577657699585, 2.1543972492218018, 0.5470017790794373, -1.3908172845840454, -0.49939751625061035, -0.13197116553783417, -1.2228492498397827, 0.44539424777030945, 1.214662790298462, -1.01875901222229, -1.8694061040878296, -0.7089757323265076, -0.3334778845310211, -1.3554775714874268, -0.4574194550514221, -0.7959914803504944, 0.018382318317890167, 0.4006015360355377, 1.6833164691925049, -1.4981781244277954, 0.4746446907520294, 0.48578545451164246, 0.4162740707397461, 1.7920111417770386, -0.2212820202112198, -1.23312509059906, 1.2871934175491333, 0.7229412794113159], [0.41054660081863403, 0.6244369745254517, 0.040655795484781265, -1.2765958309173584, -0.3137800693511963, 3.0918869972229004, -1.2810300588607788, -0.2515995502471924, -1.3466506004333496, -0.296945184469223, 1.0413116216659546, -0.28011369705200195, 1.7076259851455688, -1.6106983423233032, 0.5294298529624939, 1.2226316928863525, -0.7871044874191284, 0.45052868127822876, -1.8148610591888428, 0.49693167209625244, 0.5303730964660645, -1.8943560123443604, -0.5831460952758789, -0.21868664026260376, 1.2285218238830566, 2.5266292095184326, -0.9471998810768127, 1.5115374326705933, -2.355252981185913, 1.5124058723449707, -0.2395734041929245, 0.17399580776691437, -0.11650168150663376, 0.5642609000205994, -1.0942081212997437, 1.0086400508880615, -0.23595954477787018, 1.0161832571029663, -0.4147227108478546, 1.1561354398727417, -1.951724648475647, -0.011941681616008282, -1.5085666179656982, -1.1410621404647827, 0.9082664847373962, 1.6500827074050903, -0.8402462005615234, 1.221411108970642, -0.011470909230411053, -0.9167001843452454, 0.32044222950935364, 2.6684701442718506, -0.08164473623037338, -0.9733769297599792, -0.7798989415168762, 1.297793984413147, 1.1350525617599487, 0.3664298951625824, 0.45566263794898987, 1.2134627103805542, -0.03532399237155914, -0.42822566628456116, 1.342434287071228, 1.6892526149749756], [-0.12565013766288757, -0.47925707697868347, -0.46000936627388, -0.8518704771995544, -0.5513461232185364, 1.0885483026504517, -0.2155100405216217, 0.46254321932792664, 1.1576088666915894, -0.6094764471054077, -0.7879744172096252, -0.7509159445762634, 0.3818129003047943, 1.177822232246399, 0.5664801001548767, 1.2273292541503906, 0.48551759123802185, -1.453243613243103, -0.41415897011756897, 0.8615351915359497, -0.6955032348632812, -0.27566635608673096, -0.06874848902225494, 1.2451390027999878, -0.38241347670555115, 0.3208058476448059, -0.46620112657546997, -1.8063163757324219, -0.07408007234334946, -1.224057674407959, -0.8682390451431274, -1.3338946104049683, 0.04149336367845535, 0.9949064254760742, 2.105703353881836, 1.60672128200531, -0.9378330111503601, 0.5551780462265015, 0.8093047142028809, 1.6703741550445557, 0.6918991208076477, 0.6062118411064148, -1.052864909172058, 0.858841061592102, -0.1744404435157776, 1.0939360857009888, -1.2168718576431274, 0.919455349445343, -0.6245653629302979, 0.22285421192646027, 0.2113185077905655, -0.7526684403419495, -0.41195547580718994, 0.7127153873443604, 0.5148829817771912, -0.5728989839553833, 1.506393551826477, 1.0912630558013916, 1.130973219871521, 0.1758333444595337, 1.577433466911316, 0.10271348059177399, 0.41628706455230713, 0.32884231209754944], [-1.0910284519195557, -0.6810480952262878, -0.29461926221847534, 0.427649587392807, -0.5820872187614441, 0.5839722752571106, 1.1703993082046509, 1.4940663576126099, 0.7888424396514893, 0.48996493220329285, 0.627804696559906, -0.2891721725463867, -0.3812193274497986, -0.754464328289032, -0.41103750467300415, -0.33001452684402466, 0.09716245532035828, 1.2186238765716553, 0.7848864197731018, 0.3606616258621216, -0.809581995010376, 1.3853917121887207, -1.4347659349441528, -0.9472021460533142, 1.6988011598587036, 2.210118532180786, -0.6301257610321045, 1.693307638168335, 1.2268046140670776, 0.48938077688217163, 1.0523011684417725, 1.752219557762146, -1.6613142490386963, -0.9973418712615967, 0.303886741399765, 0.36118295788764954, 0.3805336654186249, 0.9916043281555176, 0.9228052496910095, 1.7144520282745361, -0.04221628978848457, -0.20024824142456055, 1.2169381380081177, 0.7759560346603394, -0.8580347895622253, 0.36924293637275696, -1.6891038417816162, 0.8187082409858704, -1.2797893285751343, 0.5502417087554932, 0.7027474045753479, -0.4072694778442383, 0.4440214931964874, 1.223214864730835, -0.293845534324646, 0.85865318775177, 0.28714191913604736, 1.3094677925109863, 0.31148383021354675, 1.7113282680511475, 1.6846827268600464, 0.19134168326854706, 1.409720540046692, -0.5602556467056274], [0.2733146846294403, 0.33675312995910645, -0.4310669004917145, -0.5353068709373474, 0.41884955763816833, 1.003578782081604, 0.02807454578578472, -1.786801815032959, -2.3775243759155273, -1.3046435117721558, 1.3738785982131958, -1.7121553421020508, 1.8406585454940796, -0.07604877650737762, 1.443725824356079, 1.66862952709198, 1.0335391759872437, 0.08642882853746414, 0.46084290742874146, 0.609143078327179, 0.013721984811127186, 0.006145329214632511, -0.8551350235939026, 0.93137127161026, 0.7723921537399292, 0.3186066150665283, -1.0765986442565918, 0.5105425119400024, 0.43703964352607727, 0.42264634370803833, -1.614114761352539, -2.150290012359619, -1.1753566265106201, 0.6849023103713989, -1.1405460834503174, -1.6475486755371094, 2.8191401958465576, 2.032759428024292, 1.0495355129241943, 0.3276415169239044, 0.9834493398666382, 0.32788926362991333, 0.5104051232337952, 0.8992362022399902, -0.3875962495803833, -0.7033525705337524, 1.5417035818099976, 0.8469973802566528, 0.716438889503479, -1.256806492805481, 0.7654590010643005, 0.49396267533302307, 0.1483292281627655, -0.47441622614860535, -0.3275487422943115, 0.5041687488555908, 1.1281379461288452, 0.7095222473144531, 0.5803142786026001, 0.2719651460647583, 2.238084554672241, 0.36367732286453247, 0.5376766920089722, 1.3203333616256714], [2.682798385620117, 1.1543610095977783, 0.24741877615451813, 1.0040868520736694, -1.3067620992660522, 0.10659436136484146, -2.25212025642395, 1.1021589040756226, 0.6806278228759766, 0.1306672990322113, 0.5788754820823669, 0.6155888438224792, -1.6075342893600464, -1.6931657791137695, 0.8971264958381653, -1.1248306035995483, -0.8962514400482178, 0.8194222450256348, -0.2959558367729187, -0.6506388783454895, -1.4736028909683228, -2.4263293743133545, -0.3031889796257019, 0.2771082818508148, -0.7363154292106628, -0.2086016982793808, 0.21747379004955292, -0.7550818920135498, -1.256095051765442, -0.03371400758624077, 0.7023900747299194, 0.16036102175712585, -1.1748943328857422, -0.9250844120979309, -1.4182759523391724, 0.6199857592582703, -1.219645619392395, 0.5618604421615601, 1.1805509328842163, -0.6128658652305603, -1.3095896244049072, -0.7020530104637146, -0.5390225052833557, -0.1930498480796814, -0.09900879859924316, -0.23101727664470673, 1.9592174291610718, -1.5500688552856445, -1.0020074844360352, 2.0300989151000977, 0.29536861181259155, -0.11667802184820175, -0.22296270728111267, -0.5632572174072266, -0.12952013313770294, -0.76231449842453, -0.8327909708023071, -0.7151422500610352, 0.9330930113792419, 0.2020140141248703, -1.3337219953536987, -0.6666591167449951, -0.45561373233795166, -0.3398422598838806], [0.8611280918121338, 1.024314045906067, -0.7509826421737671, 0.1463109403848648, -0.47084134817123413, -0.9033157229423523, 0.8210179209709167, -1.3378218412399292, -0.9105680584907532, -0.7147942185401917, 0.13676817715168, -0.7575318217277527, -0.3435543477535248, -0.7134798765182495, -1.3433575630187988, 1.0712313652038574, -0.24660439789295197, -0.1663249433040619, -2.4935925006866455, 0.7996471524238586, 1.296249508857727, 0.5390326976776123, 1.171697974205017, 0.806126058101654, -0.9791902899742126, 0.737190842628479, 0.29202884435653687, 1.7062773704528809, 0.11000331491231918, 0.47350531816482544, -0.7309873104095459, 1.1883183717727661, 0.10325443744659424, 1.837007999420166, -0.502129852771759, 0.04056205227971077, -1.6515458822250366, -1.238028883934021, -0.06313981115818024, 0.8089039921760559, 0.5785384178161621, -1.0555469989776611, -0.7695601582527161, -1.0912569761276245, 0.7149233222007751, -0.3497348129749298, 1.0576248168945312, -0.45747140049934387, 0.7377479672431946, 1.1110670566558838, 0.6862483620643616, 0.33977994322776794, -0.8208857774734497, -0.05702028423547745, 0.45598340034484863, -2.406998634338379, 0.9778606295585632, 0.9694875478744507, -0.03673204407095909, -0.1668352335691452, -0.7677940726280212, -0.4755761921405792, 0.016612350940704346, 0.2129596620798111], [0.7659022808074951, -0.03867129981517792, 1.857495903968811, 0.7415507435798645, -1.0094432830810547, -1.1696720123291016, 0.1424700766801834, -1.5923314094543457, -0.2640433609485626, -0.3183016777038574, -0.2851475775241852, 0.05437365174293518, 1.050094485282898, -1.0418930053710938, -2.121408224105835, -1.0969127416610718, -2.146481990814209, 0.37947800755500793, -1.0725526809692383, 0.2697124481201172, 0.9063783288002014, 0.7924642562866211, -1.5399481058120728, 0.15013891458511353, 2.979241132736206, -2.0014936923980713, 2.0184810161590576, 0.7861029505729675, 1.5248957872390747, 1.0181385278701782, -1.8016480207443237, 2.8701224327087402, 0.1830378919839859, 0.009157453663647175, 1.1422877311706543, -0.29761824011802673, 0.6003756523132324, -1.4028927087783813, 1.5638248920440674, -0.3850601315498352, 0.5168285965919495, -1.1740872859954834, 1.0162594318389893, -0.25010693073272705, -0.21766746044158936, -1.515833854675293, 0.18486739695072174, -0.5409709811210632, -1.028655767440796, 0.06926793605089188, 1.5151572227478027, 0.2553253471851349, 0.09168564528226852, 0.2739100456237793, -0.7322324514389038, 0.9581056237220764, -1.484351634979248, 0.9880775809288025, -0.6462902426719666, 0.5173008441925049, -1.5956315994262695, -0.10462504625320435, 0.9696666598320007, -0.03260647878050804], [-0.3612404465675354, -0.7437014579772949, -0.8685764670372009, 0.3722821772098541, -0.03433206304907799, 0.9054279327392578, 0.25498923659324646, 0.5181610584259033, 1.267999529838562, 1.0862693786621094, 1.0967810153961182, -0.20656587183475494, 0.8300164937973022, 0.1668967604637146, 0.48216113448143005, -1.217827558517456, 0.5570747256278992, 0.37347257137298584, 0.1545991450548172, 0.20624004304409027, 2.929121732711792, -0.3824477195739746, 0.23880308866500854, 1.6469974517822266, 1.7639955282211304, 0.6894791126251221, 0.7429144978523254, -0.5475508570671082, 0.9008873105049133, 1.3346294164657593, 0.13471730053424835, 0.8441727757453918, -1.1588481664657593, -0.766431450843811, -0.293605774641037, 1.4320193529129028, -0.11539384722709656, 0.25223931670188904, -1.0157779455184937, 0.5453954339027405, -1.2438589334487915, -0.6067362427711487, 0.5197631120681763, 0.22327031195163727, 1.5124552249908447, -0.21980896592140198, -0.8979771733283997, 0.09556513279676437, -0.08910583704710007, -2.4588680267333984, -1.2426894903182983, 0.831180989742279, 0.07891029864549637, -1.779343843460083, -0.2477693259716034, -1.907684326171875, 1.1316161155700684, -0.45384669303894043, 0.6632290482521057, 0.6179429292678833, -2.9822897911071777, -0.6914950013160706, 1.7677339315414429, 1.4111826419830322], [-0.2918219566345215, -0.07437052577733994, -2.0160720348358154, -1.2013516426086426, 1.4681614637374878, -0.37176498770713806, -0.36729922890663147, -0.7157317399978638, -0.2240152657032013, -0.402243435382843, -2.875331401824951, -0.3758802115917206, -0.5342352390289307, 0.32701173424720764, -0.1291186660528183, 0.07994641363620758, -0.13345444202423096, -1.7385807037353516, 0.40252885222435, -0.04828297346830368, 0.5128881931304932, -0.7698389291763306, -0.4370608627796173, 0.09598974883556366, -1.490476131439209, -0.09072883427143097, 1.3915337324142456, 0.16593891382217407, -0.2463529109954834, -1.6977664232254028, 0.502601683139801, -0.637067973613739, 0.100483238697052, -1.0716081857681274, -0.19683057069778442, -0.3742082715034485, 0.20680053532123566, -1.1639235019683838, 0.42707324028015137, -0.7397201657295227, 0.7518007755279541, 0.1641918420791626, 0.4939131438732147, 0.08681642264127731, -0.36659446358680725, -1.8582651615142822, 0.04926268756389618, 0.46667250990867615, 2.260632038116455, 0.5524598956108093, 1.1215062141418457, -0.03290775045752525, -1.9868494272232056, 1.1119494438171387, 0.05660402029752731, -0.006920395419001579, 0.42526209354400635, 1.87494957447052, 0.5292790532112122, 0.8355700373649597, 0.22152988612651825, 1.157779335975647, -0.7115678787231445, 0.6669341325759888], [0.25324684381484985, 0.24136781692504883, -1.065068244934082, -0.1789655089378357, -0.8108016848564148, -0.49318385124206543, -0.0011300561018288136, 0.7435081005096436, -0.7749835848808289, 1.5089576244354248, -1.4851157665252686, -0.6320840120315552, -1.2053039073944092, -0.26823028922080994, 0.7458070516586304, -0.39694398641586304, -1.5341976881027222, -0.5046027898788452, -1.3952977657318115, -0.12097818404436111, -0.25339919328689575, 0.07791390269994736, -1.5075210332870483, -0.09811938554048538, 0.05424240231513977, -1.1461730003356934, 0.6139971613883972, 0.08230552822351456, 1.5507488250732422, 0.3237145245075226, 0.26721253991127014, 0.7206739783287048, 1.291098713874817, -1.4821867942810059, -0.17275470495224, -1.0631564855575562, 0.5033499598503113, -0.6702376008033752, -1.4465723037719727, -0.5040571093559265, -0.6162071824073792, 0.6175498366355896, -1.459973692893982, -1.2826213836669922, -0.9361194968223572, -1.3754072189331055, -0.3742513358592987, 0.5826190710067749, 0.369749516248703, -1.2446303367614746, 2.046605110168457, -0.8796321749687195, 0.9424861073493958, -0.1387692391872406, -1.0572900772094727, 0.027994774281978607, 0.18111786246299744, -0.12573377788066864, -1.432151436805725, -0.33531880378723145, -0.5873640179634094, -0.09012117981910706, -0.6094955205917358, 1.8228353261947632], [-0.741503894329071, 0.36930394172668457, -1.0010441541671753, 1.1687235832214355, -0.3754737675189972, 0.5737752914428711, 0.4378921091556549, 0.7081468105316162, 3.164623260498047, -0.0420784130692482, 0.25620096921920776, -2.713581085205078, 2.2193093299865723, 0.6081618666648865, 0.4490930438041687, 0.20134681463241577, -0.10959228873252869, 0.22694231569766998, -0.8064393401145935, 2.121034622192383, 1.806734323501587, 0.004522593691945076, 0.9969769716262817, 1.0266878604888916, -0.2930634021759033, -2.186876058578491, -1.627236247062683, -0.8964093923568726, -1.0664093494415283, -0.6170181632041931, 0.8536491990089417, -2.3600950241088867, 0.007331118453294039, 0.8547085523605347, 1.1755794286727905, 0.4603208303451538, 1.638797402381897, 0.15848883986473083, -1.530666708946228, -0.5099679231643677, 0.6494866609573364, -0.08747963607311249, 1.154543399810791, 0.10971558094024658, 0.9274017214775085, 1.646531581878662, 0.7725321650505066, 0.6192336082458496, 0.584661602973938, -1.0979567766189575, 1.5104411840438843, 0.554370105266571, 1.827172875404358, 1.2882053852081299, -0.5904155373573303, -0.9650263786315918, -0.2979574501514435, 0.9872625470161438, -0.7812800407409668, -0.7713645696640015, 0.809467613697052, -0.016622673720121384, 2.2491908073425293, 0.26492416858673096], [0.21065235137939453, -0.7061442136764526, 2.4582784175872803, -0.7427368760108948, -0.7087076902389526, -1.6163073778152466, -0.5437079071998596, -0.06232528015971184, -0.5905623435974121, -0.39323899149894714, -1.0141735076904297, -1.0157105922698975, -0.1297747939825058, 1.105449914932251, -0.46010592579841614, 0.10713913291692734, 0.08771740645170212, -1.1938912868499756, -0.7661879658699036, 2.407282590866089, -1.1164673566818237, -0.25814855098724365, -1.7438066005706787, 0.830281138420105, 0.3406846821308136, -0.21591268479824066, -0.02708788961172104, -0.15632222592830658, 1.068735122680664, 0.7586230039596558, 2.0739386081695557, -0.4970669150352478, 0.814307451248169, 0.6413322687149048, 0.4985755681991577, -1.8420664072036743, 0.8791975975036621, -1.746800184249878, 1.0013127326965332, -0.27578839659690857, 0.5844805836677551, -0.4623171091079712, 0.4935707151889801, -0.9456968307495117, 0.5694421529769897, 0.9729424118995667, -1.4008125066757202, 0.981559693813324, -0.407406210899353, 1.0410958528518677, -0.567561686038971, 1.0982874631881714, -0.4077270030975342, -0.023894380778074265, 0.1823592185974121, -0.738471508026123, 0.5388438105583191, -1.3987112045288086, 1.8598074913024902, -1.4801729917526245, 0.3288027048110962, 0.553063690662384, -1.1520918607711792, -0.18422777950763702], [-0.6680145263671875, -0.7361827492713928, -0.639548122882843, 1.355027437210083, 0.46556487679481506, -0.44518864154815674, -0.08622080087661743, 1.469120979309082, 0.6121445298194885, -0.3349291682243347, -0.42633765935897827, 1.447313904762268, 0.9793969988822937, 2.299180746078491, -0.05790955200791359, 0.13878567516803741, -0.022026224061846733, 0.3885825574398041, -1.3949682712554932, 0.7042062878608704, 2.093607187271118, 1.9674264192581177, -0.5390877723693848, 2.3377175331115723, -0.6928572058677673, 0.764323890209198, -0.23473267257213593, -0.3886392116546631, 0.9482172727584839, -0.44863900542259216, 0.6774762272834778, -0.9926565289497375, -0.3601466715335846, 0.7753323912620544, -0.13465289771556854, 0.19017177820205688, 0.1909056305885315, -0.1655701845884323, -0.08860635757446289, -0.8189861178398132, -0.06463714689016342, -0.26630380749702454, 0.330757737159729, 0.9543185830116272, 0.101621612906456, -1.1274187564849854, 0.9398325681686401, 0.003649470629170537, 0.31428393721580505, 0.861454427242279, 0.44630318880081177, 0.7094656825065613, 1.3449161052703857, -0.21611547470092773, -0.1529323309659958, 0.5196197032928467, -0.7892718315124512, 0.05258890986442566, -0.3275309205055237, -0.20379842817783356, -0.03475126996636391, -1.7955082654953003, -1.0583535432815552, -2.051455020904541], [1.0790858268737793, -0.0636700764298439, -0.848222553730011, -3.496739625930786, -1.1007195711135864, 0.9651269912719727, 0.8978354334831238, -0.498351514339447, 1.3700532913208008, 0.5789014101028442, -0.44914233684539795, 0.2116490602493286, -1.0796469449996948, 0.3276960849761963, -1.5556206703186035, -0.6394530534744263, 1.4467597007751465, -2.501835823059082, 1.4158554077148438, -0.01491871103644371, 0.611013650894165, -1.0110230445861816, -0.29348883032798767, 1.23183274269104, 0.6426116228103638, 1.1297410726547241, 0.8096969723701477, -1.0443434715270996, 1.39599609375, -0.5077913403511047, -0.4102969467639923, 0.3476918041706085, -0.18033644556999207, -1.1725225448608398, -0.07489480078220367, 0.6097617149353027, 0.43656739592552185, -0.6956114172935486, -0.8520781397819519, 1.0442215204238892, 0.06768318265676498, -1.1690621376037598, -1.6745845079421997, -0.060906801372766495, 0.16210325062274933, 0.9262595176696777, 0.06167727708816528, -1.741466760635376, -0.11925603449344635, 0.42512354254722595, -1.3034679889678955, -0.4795815050601959, -0.9136903882026672, 0.7671005725860596, 1.3295187950134277, 0.3804372549057007, 0.02093561924993992, 1.2981055974960327, -1.362383246421814, -2.5796191692352295, 1.16836416721344, 0.027414465323090553, 0.05045581981539726, -1.9154715538024902], [-0.8627326488494873, 1.0821058750152588, 0.3704988956451416, 1.2444300651550293, -0.44097205996513367, 0.30574485659599304, 0.474161297082901, 0.4137551486492157, -0.32071730494499207, -0.22547218203544617, -1.1511720418930054, -0.20025625824928284, 1.467349886894226, 1.8299050331115723, 1.6405024528503418, -0.6818820238113403, 0.4150971472263336, -0.4940364956855774, 0.25740137696266174, -1.144774317741394, 2.2818918228149414, 1.4285825490951538, -0.16096682846546173, 0.22418513894081116, -0.16336600482463837, 0.04640096798539162, 0.3099956512451172, 0.5840884447097778, 1.64247727394104, 0.8215161561965942, -1.5823965072631836, -1.6078534126281738, -0.01630600541830063, -0.8691930174827576, 0.9144094586372375, 1.287498116493225, 0.5803655982017517, 1.6476489305496216, -0.38086602091789246, -0.5134628415107727, -1.760048270225525, -0.43875688314437866, 1.2758485078811646, -0.5736410617828369, -1.0975764989852905, 1.5816679000854492, -0.21195079386234283, -1.2822952270507812, 0.5018250942230225, 0.8900195360183716, -1.0951210260391235, 1.1400495767593384, -0.2183942049741745, -0.8395190834999084, -1.1383246183395386, -1.011283278465271, -0.82220858335495, 1.615820288658142, 2.2224912643432617, -1.4356720447540283, 2.7027108669281006, -1.8435516357421875, -0.12157738953828812, 1.2660496234893799], [-0.14065858721733093, -0.9836651086807251, -0.7420697808265686, -1.6343951225280762, 0.4544079601764679, -0.5536975264549255, 0.9011955261230469, -1.7399675846099854, -0.42111971974372864, 0.2080264538526535, 1.0968081951141357, 0.688113272190094, 0.5885326862335205, 0.7254340648651123, -1.3029074668884277, -0.7084510922431946, -0.9941504597663879, -0.6947202086448669, 0.6252886056900024, 0.6652932167053223, 0.6721571683883667, 1.7669341564178467, -0.9829652309417725, 0.11021027714014053, -0.5016642808914185, 0.34625282883644104, 0.9579797387123108, 0.1279553920030594, -0.15899671614170074, 0.4067438840866089, -0.19081644713878632, -0.5786523818969727, -0.5762817859649658, 1.0370582342147827, -0.5007773041725159, -0.4715922772884369, 0.6565234065055847, -1.762618064880371, 0.2435467541217804, -1.0776993036270142, 1.2367324829101562, -0.67680823802948, 0.492585152387619, -0.9454301595687866, 0.25736671686172485, 1.2073661088943481, -0.39819103479385376, -2.031229257583618, 0.9439728260040283, -0.5093928575515747, 0.14378079771995544, 0.14549334347248077, -1.6736193895339966, 0.049301877617836, 0.29021546244621277, 0.17454646527767181, 0.6687688231468201, -0.36427345871925354, 0.47417721152305603, 0.6816691160202026, -0.7456821799278259, 0.14543402194976807, -0.9888933300971985, 1.5190316438674927], [-0.41462111473083496, 1.8848562240600586, -0.09421364217996597, 1.1588575839996338, -0.9733827114105225, -0.8270201683044434, 1.5428072214126587, -0.43308666348457336, 0.15189479291439056, 0.9949327111244202, 0.04870148375630379, -1.36822509765625, -0.8687079548835754, -2.074228048324585, 1.253617525100708, 0.3130798041820526, 0.6068499088287354, -0.6975377202033997, 0.38146543502807617, 0.6144384741783142, -0.562207818031311, 0.1637178510427475, -0.3791807293891907, -0.2920031249523163, -2.0926430225372314, -0.09610657393932343, -0.4817756414413452, 0.3126504123210907, -0.5537451505661011, -0.9879850149154663, -1.40598726272583, 1.0781618356704712, 1.3709582090377808, -0.3709114193916321, 0.4873116910457611, -0.7690775394439697, -0.3709120452404022, 1.3806610107421875, 0.9310237169265747, -1.131338119506836, 1.3147027492523193, -0.413173109292984, -0.20013803243637085, 1.9117958545684814, 0.8248516321182251, -0.7959629893302917, -1.4891712665557861, -0.925104022026062, 0.8011273741722107, 1.093645453453064, -0.11584985256195068, -0.6586158275604248, 0.26344725489616394, -1.6600948572158813, 0.8048877716064453, 0.8445559144020081, 0.5704486966133118, -0.15404832363128662, 1.0865936279296875, 0.22110292315483093, -1.182804822921753, 0.975256085395813, -1.8746130466461182, 0.8151569366455078], [-1.0084147453308105, -0.0013191907200962305, 0.34705686569213867, 0.1704319566488266, 0.031152354553341866, -1.4298439025878906, 0.9349915385246277, -1.8677167892456055, 0.5249056816101074, -0.129958376288414, -0.46870169043540955, 0.611131489276886, -0.11428727209568024, 1.779043436050415, -0.19024285674095154, -0.3641151487827301, -0.5056746602058411, 0.5283396244049072, -0.9192965030670166, -0.50555419921875, 0.668808102607727, 1.4738686084747314, 1.380256175994873, 0.03166613727807999, -1.464720606803894, 0.17958442866802216, -0.5594843626022339, 0.5288706421852112, -1.2934156656265259, -0.7149851322174072, 0.4086938500404358, 1.0792888402938843, 1.0295904874801636, -0.34735819697380066, 1.3473381996154785, -0.27860113978385925, 1.465185523033142, 0.18749158084392548, 1.9941469430923462, 0.17634916305541992, 1.7869101762771606, -1.8798989057540894, 0.06386147439479828, 0.8398619890213013, -0.765636682510376, -0.5177478194236755, -0.3301073908805847, -1.7148799896240234, 0.6662822961807251, -0.24059058725833893, 1.6786164045333862, 0.987089991569519, 0.26374825835227966, -0.07010466605424881, -1.4036544561386108, -0.33729469776153564, 0.20412074029445648, 0.41452962160110474, -0.4031282663345337, -1.6234062910079956, -0.07597284764051437, 1.5111116170883179, 0.29031267762184143, -0.7471922636032104], [0.9490533471107483, 0.34846654534339905, 1.2382627725601196, -0.8966127634048462, 0.034195464104413986, 0.32216987013816833, -1.3364222049713135, -0.10389113426208496, -0.13791371881961823, -1.0890522003173828, 1.615325689315796, -0.3137093186378479, -0.3337218165397644, 0.587741494178772, -0.10475566983222961, 0.4680171310901642, -0.8925409317016602, -1.5732725858688354, -0.618928074836731, -0.7701961398124695, 0.5603052973747253, -0.5757522583007812, 1.2317287921905518, 1.0685656070709229, -0.2677481472492218, 0.26003044843673706, 1.6487700939178467, 0.4972214102745056, -0.46869155764579773, -0.442645400762558, 1.5546199083328247, 1.5964031219482422, -0.5444139838218689, 0.1765376329421997, -0.13695479929447174, -0.6718408465385437, 1.5292922258377075, -0.5795369744300842, -0.1184845119714737, 0.8406778573989868, -0.042464569211006165, 0.16741059720516205, -0.17550215125083923, -0.2877820134162903, 1.2425167560577393, 0.35506850481033325, -0.36411336064338684, -0.12799760699272156, 0.7471523880958557, 0.43694838881492615, 1.3210968971252441, 0.1530001014471054, 0.356362909078598, 0.7228000164031982, 1.1302392482757568, -1.0540056228637695, 0.6725332736968994, 0.7223231196403503, 0.9116823673248291, 1.2486116886138916, 1.4081398248672485, -1.9488496780395508, 0.13502930104732513, -1.0377017259597778], [0.6497159004211426, 0.36397403478622437, 0.768186092376709, -0.8739001750946045, 0.4388592541217804, 1.4183101654052734, 1.6716824769973755, 0.3970924913883209, 0.4395548105239868, 0.3418317139148712, 1.1514432430267334, -0.9422274827957153, 1.6904950141906738, 1.078934907913208, -0.827597439289093, -0.3570155203342438, -0.3042220175266266, -1.2100181579589844, 1.5867277383804321, 0.2779967486858368, 1.243541955947876, 0.185747429728508, 0.7413432598114014, 0.8870947360992432, 0.3436107933521271, 0.9564417004585266, 0.9329180717468262, 1.4139533042907715, -0.1472022831439972, -1.1534620523452759, -0.565631091594696, -0.636046826839447, 0.3497627377510071, -0.405350923538208, -1.097271203994751, -0.8470376133918762, -0.7113251686096191, -2.2823739051818848, 0.2303532063961029, 0.3857623040676117, -0.44775068759918213, -0.0029310015961527824, -0.5950545072555542, -0.7496829032897949, 1.2063484191894531, -0.08037443459033966, -0.978803277015686, 1.0455623865127563, 0.5857494473457336, 0.06598029285669327, -0.7995694875717163, -0.439448744058609, -0.6718198657035828, 0.4716806709766388, -1.3258395195007324, -1.7515250444412231, -0.2451251596212387, -0.5363542437553406, 0.7406294941902161, 0.36269867420196533, 0.8190075159072876, -1.0457074642181396, -2.012693166732788, 1.557625412940979], [-0.761235773563385, 0.1529676616191864, -0.2149270623922348, -0.2912173271179199, -1.86319899559021, -0.8561685085296631, -2.0868468284606934, -1.0269384384155273, -1.777999758720398, -0.6012528538703918, -0.5389994978904724, -0.9306659698486328, -0.5847219228744507, -0.48282837867736816, 0.9736027121543884, -0.5521314144134521, -1.1280531883239746, 0.7652178406715393, -0.05535244196653366, 0.59026700258255, -0.463198721408844, -1.6525746583938599, -0.5304173827171326, -0.3181081712245941, -1.3391923904418945, -0.9468777179718018, 0.24759958684444427, 0.643539309501648, -1.675048589706421, 0.8764575123786926, -0.8746156096458435, -1.3518781661987305, -1.004231572151184, -0.13541917502880096, 0.164182648062706, 0.8355482220649719, -0.9157053232192993, -0.48509421944618225, -0.49885985255241394, 0.6719857454299927, -1.2525856494903564, -0.31435367465019226, -0.5255485773086548, 0.3222558796405792, -0.651729941368103, -0.7232498526573181, -0.20240753889083862, 0.6344277858734131, 0.08265058696269989, 0.11698711663484573, 0.6587986946105957, -0.893618106842041, 0.3275293707847595, -1.107828974723816, -1.9410697221755981, -0.7116909623146057, -0.6418048739433289, 0.04615357145667076, 0.24975737929344177, 0.6001772880554199, -1.1900804042816162, -1.3755544424057007, 1.720870018005371, -2.6076841354370117], [-0.16888347268104553, 0.22956077754497528, -0.4007163643836975, -0.22873961925506592, 0.6020514369010925, 0.7085375189781189, -1.2689430713653564, 0.402508407831192, 0.1654372215270996, -1.001103401184082, -0.16432055830955505, 0.022325940430164337, 0.8425105214118958, -2.6392288208007812, 0.7437794208526611, 0.29839175939559937, 0.401294082403183, 0.08057788014411926, -0.2682655453681946, 1.5444432497024536, 0.5978733897209167, -0.393036425113678, -0.17401652038097382, -0.22473730146884918, -0.15969307720661163, -1.0540663003921509, -1.613635778427124, -0.38354426622390747, -0.2809670865535736, -0.9227701425552368, 0.7307320833206177, -1.3021466732025146, -0.04376747086644173, 1.4652700424194336, 0.47274526953697205, 0.5907618999481201, -1.289624571800232, 0.39404183626174927, -0.8536776304244995, 0.5076780915260315, 0.48863115906715393, -0.7578667998313904, -1.8010313510894775, -0.4648473560810089, 0.6634225845336914, 0.6754546761512756, -0.6384646892547607, 0.34931862354278564, -0.7632729411125183, -0.9520545601844788, 0.7546859979629517, -0.093141108751297, -0.18069487810134888, -0.82756507396698, -0.6983396410942078, 1.7163894176483154, 0.8303170204162598, -0.0404122993350029, 0.6868900656700134, -0.08526836335659027, 0.29495665431022644, -1.2965638637542725, 1.1320922374725342, -2.350628614425659], [-0.10686766356229782, -0.2433195859193802, 0.47115665674209595, 0.005490703508257866, -0.6970155239105225, -0.040239058434963226, -0.31085342168807983, 0.8983352184295654, 1.0013368129730225, 0.6038374900817871, 0.5794508457183838, -0.5902421474456787, 0.3033856153488159, -1.7248270511627197, -1.3388675451278687, -1.2851741313934326, 0.02074624039232731, -1.243621587753296, -0.6601951122283936, -0.6580162048339844, -0.6636818051338196, -0.9231103658676147, -0.4576595425605774, -1.3162611722946167, -1.8327558040618896, -1.7475899457931519, -0.13451793789863586, 3.0271010398864746, 0.09473904967308044, 0.13218359649181366, 0.33525893092155457, 1.1228835582733154, -0.052757829427719116, -0.6808229684829712, -0.013350304216146469, 0.23892322182655334, -0.8412392735481262, 0.5844506621360779, -0.7795257568359375, 1.161475658416748, 0.30390626192092896, 0.2987213730812073, 1.015614628791809, -1.608994722366333, -1.1341309547424316, 0.1112254410982132, -1.498022437095642, -1.2383229732513428, 0.3781200647354126, 0.3363261818885803, 1.1053844690322876, 0.273431658744812, -0.43383926153182983, 1.5771673917770386, -0.19864733517169952, 0.5569968223571777, -2.496567964553833, 0.623748242855072, -0.10883013904094696, 0.36689886450767517, -0.30034029483795166, 1.8158533573150635, -0.8210768699645996, 1.610000729560852], [0.2016015499830246, -1.166249394416809, 0.7738175392150879, 0.41156062483787537, 1.0234781503677368, -2.3743577003479004, -1.7828400135040283, -0.3465026915073395, 0.26724767684936523, -0.3155858814716339, 1.9303579330444336, -0.06760476529598236, 0.4299369752407074, -0.2849401831626892, -0.46239736676216125, -0.36949416995048523, -0.7079477310180664, 0.3567909598350525, -0.12333253771066666, 1.3094614744186401, -1.145604133605957, 0.8092211484909058, -1.7290714979171753, 0.7900136113166809, 1.1501752138137817, -0.5777007341384888, 3.5613739490509033, 1.3061740398406982, 0.8964909911155701, -0.08722803741693497, -0.49614042043685913, 0.958248496055603, 0.8602233529090881, -1.965760588645935, 0.9235294461250305, -2.2271578311920166, 0.5747610330581665, 1.0227700471878052, -0.715227484703064, 0.5969293117523193, 0.30374765396118164, -1.4735748767852783, 0.1402273029088974, -1.0423609018325806, 0.8921065926551819, 0.4813394248485565, -0.8782944679260254, -0.14647041261196136, -0.5098234415054321, 0.6511448621749878, 0.12718544900417328, -1.1135903596878052, -0.5757835507392883, -0.051214784383773804, 0.19140975177288055, 0.2446800023317337, 0.8422605991363525, 0.05615559592843056, 0.46003153920173645, -0.3351714313030243, 0.5939040184020996, 1.470819354057312, 0.10646270960569382, -0.5546694993972778], [0.5237879753112793, 1.3856536149978638, -0.6214033365249634, 1.2960435152053833, 1.1737887859344482, -0.36828410625457764, 1.6561998128890991, 0.5436393618583679, -0.9334803223609924, -0.9582352042198181, 1.8607373237609863, -1.871572494506836, -0.688456118106842, 0.18087084591388702, -1.910484790802002, -0.44185298681259155, 0.9137914776802063, -1.977063536643982, -0.6079026460647583, -0.44087791442871094, 0.6490601301193237, 0.5645822286605835, -2.2004613876342773, 0.08761085569858551, -1.368585228919983, 0.1176183819770813, -0.9557337760925293, -0.2727753520011902, -0.04093597084283829, 1.3133422136306763, 0.5359783172607422, -0.2356923669576645, -0.46507319808006287, 0.09413060545921326, 0.10269629210233688, 0.010320909321308136, -1.4341952800750732, 2.2352325916290283, -0.18620344996452332, -0.01612764038145542, 0.5452673435211182, -0.41847723722457886, 1.0336617231369019, -0.01632688194513321, -0.3867238759994507, 0.36122408509254456, 1.057711124420166, -0.7506144642829895, -0.515093207359314, -0.046523936092853546, -0.5539655685424805, 0.43500521779060364, -1.3398406505584717, -0.6378353238105774, -1.5292158126831055, -0.22863776981830597, 0.8500752449035645, 0.052901748567819595, -2.143617630004883, 1.9187179803848267, -0.9650037288665771, 0.6016373634338379, 0.43779394030570984, 0.43486469984054565], [-0.4049725830554962, 0.24148571491241455, 0.3497435748577118, 1.2253718376159668, -0.2970028519630432, 0.4304054081439972, 0.23745593428611755, 1.1859346628189087, 1.3732237815856934, 0.4817204773426056, -0.8843827247619629, -0.6378291845321655, -0.6405483484268188, -0.40849414467811584, -1.3753575086593628, -0.3852391541004181, -2.054105281829834, 0.846538245677948, -0.7489492297172546, -0.4298016428947449, 2.1172375679016113, 1.4194540977478027, 0.9563995599746704, -0.9966990947723389, 2.110475540161133, -0.5308181643486023, -0.35206031799316406, -0.5059014558792114, -0.7324122190475464, -0.2895359992980957, -1.3520255088806152, 0.3286030888557434, 0.2020939737558365, 0.25174447894096375, 0.4517744183540344, 0.3121146261692047, -0.038215186446905136, -0.47284018993377686, 0.3022922873497009, -0.3661707937717438, 0.16021762788295746, 1.3958903551101685, 0.7319891452789307, 1.9483994245529175, -0.43961989879608154, -0.4534291923046112, -1.1461400985717773, 0.1506310999393463, 0.3027222454547882, 0.8907101154327393, -0.479547917842865, -2.270932912826538, -0.07223884016275406, -0.8187171816825867, 1.0358269214630127, 0.02454676851630211, -1.8454291820526123, 2.2016842365264893, -0.7089025378227234, -2.066155195236206, -0.096694715321064, 0.109630286693573, 0.41779857873916626, -2.0658440589904785], [0.38916146755218506, 0.5579811930656433, -1.1571059226989746, 0.08460278064012527, -1.0053709745407104, 1.045555591583252, -1.5474841594696045, -0.7689399123191833, 0.6412375569343567, 1.111703872680664, 1.134325385093689, 0.5455910563468933, -0.17149591445922852, 2.154860734939575, -0.4401101768016815, -0.5563513040542603, -1.4619287252426147, -0.17700983583927155, 0.22769306600093842, 1.3483991622924805, 0.4130899906158447, 0.7126333713531494, 0.5713827610015869, -1.0824193954467773, 0.8329146504402161, -1.376570224761963, -0.2735065221786499, -0.13133321702480316, -1.127389907836914, -0.15668359398841858, 0.48814642429351807, -1.52784264087677, -0.6837963461875916, 0.117582768201828, -0.9867669343948364, -0.3115118145942688, 0.8244286179542542, -1.9079654216766357, 0.699665904045105, 0.11579729616641998, 1.290921688079834, -1.803197979927063, 1.774094581604004, -0.6852564811706543, 0.19071438908576965, -0.4251561462879181, 1.4902708530426025, 0.11449927091598511, -0.7154887318611145, -1.2537052631378174, -2.463785171508789, -0.7217411398887634, 0.9667536616325378, -0.5342167019844055, -1.259546160697937, 0.5206980109214783, 1.7841862440109253, 0.8403568863868713, 0.020385542884469032, -0.4281827509403229, 1.4551507234573364, 0.4265593886375427, 0.49767038226127625, 1.3393104076385498], [0.6592368483543396, 0.9497199058532715, -1.3328430652618408, 0.36315658688545227, -1.104721188545227, 0.8355710506439209, 0.3712257146835327, -1.487646222114563, 1.507547378540039, 0.821237325668335, -0.4616927206516266, 0.01194161269813776, 3.081784248352051, 0.8943060040473938, 0.6997042298316956, -1.736968994140625, 0.22557099163532257, -1.2846871614456177, 0.0015366615261882544, 1.2657808065414429, -1.0868823528289795, 0.42526504397392273, 0.07134935259819031, 0.8402854204177856, 0.7754358053207397, 1.0277332067489624, -0.7911194562911987, 1.967140793800354, -0.432524710893631, -0.29330962896347046, -2.016469955444336, -1.4695206880569458, 0.2798541486263275, 1.5991463661193848, 1.067226767539978, -0.020635876804590225, 0.6404042840003967, 0.13423380255699158, -1.5814915895462036, -1.8483319282531738, 0.8764765858650208, 0.21013136208057404, 0.13404205441474915, -0.3812474310398102, -1.5709433555603027, 0.43467476963996887, 0.9858474135398865, 0.2409607470035553, -0.29772159457206726, 1.0452312231063843, -0.797900915145874, 0.7174824476242065, 0.19644956290721893, -0.14967648684978485, -1.1986374855041504, 0.08842895179986954, -0.05035651847720146, 2.1487858295440674, -0.037564925849437714, 0.7735411524772644, 0.5512554049491882, -1.5982754230499268, -0.974509596824646, -1.3760937452316284], [-0.49714332818984985, -1.0441759824752808, 0.5116959810256958, -1.2674534320831299, -1.1155186891555786, 1.0161691904067993, 2.987029790878296, -0.8732265830039978, -1.2856009006500244, 0.26723238825798035, 0.3597010374069214, 0.6250424385070801, 0.02452874928712845, -0.3168465793132782, -0.9478220343589783, 0.46563026309013367, -1.5809428691864014, 0.4708161950111389, -0.6600214242935181, 0.7395182847976685, -0.18150988221168518, 0.8854843378067017, -0.8371604084968567, 1.419791579246521, 0.6736201643943787, -0.5925959348678589, -0.5115509033203125, 1.4251543283462524, -2.2101213932037354, -0.04901507496833801, 0.13041432201862335, -0.141665980219841, -1.1714848279953003, -0.47929418087005615, 0.9454832673072815, -0.050887007266283035, 1.1742416620254517, 0.30552050471305847, 0.5030154585838318, 0.027494065463542938, 0.4288122057914734, -1.2080808877944946, -1.246180534362793, 1.058038353919983, -0.905489981174469, -0.2874615788459778, -1.3696175813674927, 0.4212721586227417, 0.3552899956703186, -1.5100940465927124, 0.22952726483345032, -2.6023190021514893, 1.216107964515686, 1.0911502838134766, -0.9301717281341553, 0.1262054443359375, -1.4298570156097412, 0.24116180837154388, 0.20316970348358154, 0.7402478456497192, 1.8016836643218994, 2.746689796447754, -0.07759859412908554, 0.6051437258720398], [-1.5620893239974976, 0.5438789129257202, 1.1394387483596802, -0.32851099967956543, 0.5588134527206421, -1.5879260301589966, -0.504817008972168, -0.13920074701309204, -0.2987153232097626, 1.7580939531326294, 0.8731194138526917, -0.1123400554060936, 1.7728770971298218, 0.710101842880249, -1.2330286502838135, -0.04239778220653534, -0.3617388606071472, -0.09061839431524277, 1.5490922927856445, 1.5285221338272095, 0.23472575843334198, -1.1035553216934204, 1.4156512022018433, 0.03338855877518654, 1.6409577131271362, 0.18838077783584595, 0.5992078185081482, -0.7680897116661072, -0.8810076117515564, 1.219818115234375, 1.4888161420822144, 1.1924632787704468, -0.24793103337287903, -1.285313367843628, 0.008739122189581394, -0.3976222574710846, -1.5244700908660889, 0.5982264280319214, 0.2969367802143097, 1.8811137676239014, -0.4393315315246582, -0.07243361324071884, 0.4307537376880646, -1.2991130352020264, -0.8205446600914001, -1.136218786239624, -0.17734238505363464, 1.0246000289916992, -0.9524304270744324, 2.71386981010437, -0.9197431802749634, 0.6134259700775146, -1.4606499671936035, 1.3637908697128296, 1.188753604888916, -1.2211003303527832, -0.3157957196235657, 0.8659793138504028, 0.1927637755870819, 0.25332871079444885, -0.601906955242157, -0.08335482329130173, 0.5071074366569519, 1.0521798133850098], [0.6783718466758728, 0.5150684118270874, -0.8990784287452698, 0.2574327886104584, 0.666423499584198, -0.775894820690155, -0.45148608088493347, 0.6766011714935303, 0.18062113225460052, 0.1489560604095459, -0.532436728477478, -0.6057082414627075, -0.9336243867874146, 1.1627517938613892, -0.058624107390642166, -2.5524630546569824, 0.3654899001121521, 1.0957461595535278, -0.4127516746520996, -1.2690250873565674, 0.20085352659225464, -0.11172930151224136, 0.8281837701797485, 0.15198558568954468, -1.5013153553009033, 0.801598310470581, -0.8390493392944336, 1.2652312517166138, -0.6738548874855042, -0.14586268365383148, -0.04294488579034805, -1.2869963645935059, -0.24229031801223755, 0.28758567571640015, -0.03861204907298088, 0.2378402054309845, -0.5659772157669067, -1.360779881477356, 0.14310510456562042, -0.30601122975349426, 0.9995903372764587, -0.21469536423683167, -0.2636954188346863, -1.453881025314331, -1.2415767908096313, 1.0258338451385498, 1.8070167303085327, -0.7246366143226624, 0.3123863637447357, 0.34035688638687134, -1.837354063987732, 0.2151949405670166, 0.3273187577724457, -0.9698058366775513, 0.012248505838215351, -0.5144546627998352, 0.6095823049545288, 0.8387430906295776, 0.7769259214401245, -0.6212754249572754, -0.7807248830795288, -0.45203179121017456, 0.5148680210113525, -0.7615104913711548], [-0.13849171996116638, -0.16729266941547394, -1.4806658029556274, 0.8344802260398865, 0.035682570189237595, 0.9021099805831909, -0.7926914691925049, 0.6583417654037476, 0.7387159466743469, 0.9360882639884949, -1.5135964155197144, 1.4181264638900757, -1.453631043434143, -0.24750745296478271, 0.12124378979206085, 0.48730722069740295, -0.4932628571987152, 0.586286187171936, -0.6183293461799622, -0.24022136628627777, 0.4774019122123718, 1.1169220209121704, 0.8603614568710327, -1.1023931503295898, 1.911910891532898, -0.7860018014907837, 0.0492364875972271, 1.0656931400299072, -0.8841094970703125, 0.2862023711204529, 1.1968153715133667, 0.3624415695667267, -2.4728260040283203, 0.2624813914299011, -0.3709917366504669, -1.0751019716262817, 0.6998291611671448, 0.8651341199874878, 0.5663582682609558, -0.7500767111778259, 0.5579073429107666, -0.34170985221862793, -0.43112245202064514, 0.9898431301116943, 1.375663161277771, 0.6041696071624756, 0.8974232077598572, -1.4423465728759766, -0.146004781126976, 0.36206501722335815, -0.6294392943382263, -0.7247300744056702, 1.8961844444274902, 0.36005276441574097, 1.758089303970337, 0.8116143345832825, 0.19606944918632507, 1.225017786026001, 0.24719807505607605, 0.5166503190994263, -0.5039921998977661, 1.263968586921692, -0.2701127231121063, -1.4052703380584717], [0.004644496366381645, -0.12068377435207367, -1.5668814182281494, -0.5205374360084534, 0.5470494627952576, -1.3374508619308472, -0.9076831340789795, 1.4130940437316895, -1.1531693935394287, 1.2801512479782104, -0.17440755665302277, 1.3300518989562988, 0.873180627822876, 0.6276702880859375, 1.233638048171997, -2.1562702655792236, 0.7468004822731018, 0.10604444891214371, 0.7674656510353088, -0.39587780833244324, 1.2166115045547485, -0.047546058893203735, 0.4586966335773468, -1.4227049350738525, 0.2275945097208023, 2.1374526023864746, 0.5809855461120605, -0.7248849868774414, 1.4631849527359009, -1.033401608467102, -2.1635546684265137, -0.732636570930481, 1.2213937044143677, -0.9959121346473694, -0.11450919508934021, -1.2909437417984009, -0.7720411419868469, -0.25035765767097473, -0.990262508392334, -1.6378445625305176, -1.4287601709365845, 0.11341075599193573, 0.8799368143081665, 1.3285149335861206, 1.1876765489578247, 0.10635031014680862, 0.6679169535636902, 0.06618563085794449, 0.2735646069049835, 0.4513254165649414, 0.41901397705078125, -1.1153182983398438, 0.14133363962173462, -0.7183240056037903, 0.6352181434631348, -0.9499621391296387, -0.24978426098823547, -0.4919669032096863, -0.4503287076950073, 0.7373222708702087, 0.5888909697532654, 2.4350674152374268, -1.1886197328567505, 0.6793760657310486], [0.3563167154788971, 2.592095375061035, 0.06870831549167633, -0.05245095118880272, 1.3707562685012817, -0.895982563495636, -0.2991117238998413, 0.03818468749523163, 0.2605166435241699, -0.964653730392456, 0.43097731471061707, 0.4808904230594635, 0.11266990751028061, -0.9240456819534302, 0.671570360660553, 0.26286253333091736, -1.1406090259552002, -1.8899240493774414, -0.6836390495300293, -0.07428302615880966, -0.7749966979026794, -0.46295347809791565, 0.40591078996658325, -0.5771378874778748, 0.9606651663780212, -0.845920205116272, -2.0763325691223145, 0.874619722366333, 0.08658565580844879, -1.1761654615402222, 1.819083333015442, -1.4460346698760986, 0.06562507152557373, 0.4538077116012573, 0.14154911041259766, 0.9018211364746094, 0.25354281067848206, -0.511141300201416, 1.228920340538025, -1.1345560550689697, -0.05925430357456207, -0.5550615787506104, -1.219153881072998, 0.007644210942089558, -0.9645874500274658, -1.4124584197998047, 1.3264663219451904, -0.5944786667823792, -1.6065037250518799, -0.40353459119796753, -0.8799612522125244, -0.06454595923423767, 0.26417094469070435, 1.5733214616775513, -1.1394941806793213, 0.6806536912918091, -0.7054954767227173, -0.8440096378326416, 1.9610949754714966, 0.5988538265228271, 1.9001131057739258, -0.5411152243614197, 0.2030496448278427, -0.015655329450964928], [-0.12481115758419037, 1.9900522232055664, 0.19106966257095337, -1.2240546941757202, 0.17172540724277496, 0.36455851793289185, -0.6671581268310547, 0.7396032214164734, -0.3227662444114685, -1.898641586303711, -0.31366944313049316, -1.1882004737854004, -0.569997251033783, 0.5116829872131348, 1.482162356376648, 0.8563802242279053, 0.6906698346138, 0.9406430125236511, 1.226456642150879, 0.30606430768966675, 1.6124818325042725, 1.2600185871124268, -0.8404487371444702, -0.7530298829078674, 1.1965376138687134, 0.13519544899463654, 0.4657243490219116, -0.24577364325523376, -0.192820742726326, -0.17542503774166107, 1.5565645694732666, 1.8885376453399658, 0.6555439233779907, -0.23280270397663116, 0.5577966570854187, 0.7248297929763794, -1.293821096420288, -0.7136482000350952, -0.24480664730072021, 0.38164377212524414, -0.29623115062713623, 0.044855114072561264, -1.0351251363754272, 0.19244468212127686, -0.14216981828212738, -1.5339627265930176, -0.31281834840774536, 0.7359951734542847, 0.4725227952003479, 1.8632105588912964, 1.5405604839324951, -1.0420666933059692, -0.31417712569236755, -1.8707088232040405, -1.2394481897354126, -0.2421218752861023, -0.09015628695487976, -1.082897424697876, 2.2109851837158203, 0.2519201636314392, -0.8824571967124939, 1.285772681236267, -0.8200033903121948, -0.14950482547283173], [0.7121877074241638, 0.8972626328468323, -0.375990092754364, -0.4360258877277374, 0.35212305188179016, 0.6770857572555542, -1.451518177986145, 1.1308618783950806, -0.6528480052947998, -0.45661383867263794, -1.0834970474243164, 0.5176032781600952, -0.5216027498245239, -0.8954669237136841, 0.5867773294448853, 1.7455902099609375, 1.774274468421936, 1.1161073446273804, -0.31214091181755066, -2.3267061710357666, 0.36182093620300293, 0.474651038646698, -0.7664347290992737, 0.41148024797439575, 0.48009175062179565, -0.06188786402344704, 1.0133916139602661, 0.4883454144001007, -0.6077641844749451, -0.0246119424700737, -1.2592308521270752, 0.5213471055030823, -0.07819905877113342, -0.015180771239101887, -0.9762517809867859, 0.5566759705543518, 0.1327713131904602, 0.4479120373725891, 0.39575067162513733, 2.346251964569092, -1.6602500677108765, -0.7121894359588623, -1.7445430755615234, 0.843716025352478, 0.37258654832839966, 0.3024618625640869, 0.33519190549850464, -1.817513346672058, 0.13624875247478485, 1.0628507137298584, 1.2244888544082642, 0.4620773196220398, -0.10839042067527771, 0.1986604630947113, 0.8560271263122559, 0.8884536027908325, -0.42774003744125366, -0.7831143736839294, -1.1448527574539185, -1.2697935104370117, 1.258880853652954, 0.08170025795698166, 0.6196658611297607, -1.3904168605804443], [-0.9694796204566956, 0.02203252539038658, 0.08139389753341675, 0.5438872575759888, -1.1324479579925537, 0.48584529757499695, 0.17224621772766113, 1.2650200128555298, -1.8061141967773438, -0.7765689492225647, 0.03483354672789574, 1.5639528036117554, 0.7888898253440857, -0.3972817063331604, -1.317698359489441, -1.0360041856765747, -0.015722043812274933, 0.8480256795883179, -1.7586281299591064, 0.8153440356254578, -1.5078998804092407, -1.5702449083328247, 1.0555287599563599, 0.24666182696819305, -1.249794602394104, -0.4472799301147461, 1.7925543785095215, -1.2427948713302612, -1.6125049591064453, -0.7463582158088684, -0.40119650959968567, 0.46888354420661926, 0.3287055790424347, 1.3163845539093018, 0.6772045493125916, -2.740649700164795, -0.2553398013114929, 0.27851414680480957, -0.9349656105041504, -0.22184988856315613, -0.5117266774177551, -0.6179840564727783, -0.1254546046257019, -0.3819997310638428, 2.0034706592559814, -0.7686436772346497, -0.7949676513671875, 0.3376835882663727, -1.6687698364257812, -0.036547884345054626, 1.3057631254196167, -0.4163353145122528, 0.11367689818143845, -0.2491605281829834, 0.7149948477745056, -0.8368237018585205, 0.19157780706882477, -0.964799165725708, -0.3149721920490265, -1.4501607418060303, 0.8854621052742004, -1.4575929641723633, 0.08667907118797302, 1.4975146055221558], [0.9028213620185852, -0.5858430862426758, -0.09648363292217255, -0.7535565495491028, -0.5010713934898376, 0.45245808362960815, 0.012294015847146511, 0.35657966136932373, 0.7948764562606812, 1.3151795864105225, -1.1513036489486694, -1.551964282989502, -0.3424838185310364, 1.2153208255767822, -0.6725789308547974, -1.0188204050064087, 0.5930126905441284, 1.1948912143707275, -1.6755483150482178, -1.4465389251708984, 1.09397292137146, 1.5423619747161865, -3.6990420818328857, 0.4819594919681549, 0.5450445413589478, 0.4721829891204834, -0.5309075117111206, 0.12672393023967743, 0.5961922407150269, -0.9911426901817322, 1.3185704946517944, -1.0568474531173706, -1.0297003984451294, -1.1769471168518066, 0.0037812425289303064, -0.5051408410072327, -0.21619942784309387, 0.33091261982917786, 1.521904468536377, -0.21798966825008392, -0.6206924915313721, -0.4115421175956726, 0.11101074516773224, -1.0621942281723022, 0.20027022063732147, -2.3128507137298584, 1.1464852094650269, -1.9513647556304932, -0.12019049376249313, 0.35173681378364563, 0.15005286037921906, 0.1969713419675827, 1.3842207193374634, 0.6253087520599365, 1.3237358331680298, 0.18713180720806122, -0.1703038364648819, -0.5638731718063354, -0.24531204998493195, 1.4155796766281128, 0.18164344131946564, 0.9603356122970581, -1.1334617137908936, 0.19313523173332214], [0.5149215459823608, 0.06723863631486893, -0.29171133041381836, 2.166257619857788, 0.7378217577934265, -1.6032382249832153, 0.4538962244987488, 0.2925877869129181, 0.9234319925308228, -2.1212151050567627, 1.7462496757507324, -1.7674278020858765, 0.9312614798545837, -0.04347606003284454, 0.5755004286766052, 0.7547353506088257, 0.8276327848434448, 1.3980833292007446, -0.49861758947372437, -0.27953410148620605, -0.27662524580955505, 0.7741713523864746, 0.30142301321029663, 0.04509309306740761, -0.3105379343032837, -0.3461964726448059, -0.33001843094825745, -1.8421217203140259, -1.715935468673706, 1.4304821491241455, -1.5277154445648193, 1.4097933769226074, -0.5754877328872681, -0.16561324894428253, 2.069135904312134, 0.015995489433407784, -1.5208990573883057, 0.4875734746456146, -0.39537206292152405, 0.8731324672698975, 0.6076760292053223, 0.37247729301452637, -1.2777187824249268, 0.5275216698646545, -0.4593923091888428, 0.7586280703544617, -0.2718021869659424, 0.48845067620277405, 1.1447395086288452, -0.9862456917762756, 2.087709665298462, -0.569678544998169, 0.6338865756988525, -2.9397997856140137, -0.46591976284980774, -1.451462984085083, 0.8412827849388123, 0.02436225116252899, 0.07049494236707687, 1.2217020988464355, 0.3964082896709442, 0.7777866125106812, 0.48987263441085815, 0.8106490969657898], [1.1303569078445435, 0.5878821015357971, 0.13192026317119598, 1.4871761798858643, -1.520721197128296, 0.8151045441627502, 0.2194141447544098, 0.1671905815601349, 1.66170072555542, -0.29880163073539734, 0.11822451651096344, -0.4585299789905548, -1.2899349927902222, 1.160736083984375, 1.8810187578201294, -0.708901047706604, -1.7130144834518433, 1.4009368419647217, 0.8565692901611328, -0.5276041030883789, 1.0281040668487549, -1.0054999589920044, -0.43352341651916504, -1.4658358097076416, 0.6804315447807312, -1.3105735778808594, 1.1210765838623047, 0.7845911979675293, -1.4007858037948608, -2.165930986404419, -1.711472511291504, 1.440849781036377, -0.04935452714562416, 3.0098929405212402, 1.96397864818573, -0.5333273410797119, -1.2508865594863892, -0.8641866445541382, 0.3164512813091278, 1.3088550567626953, -0.9382851719856262, -0.8693405389785767, 1.5278539657592773, 1.0658068656921387, -1.6647125482559204, -0.5873497724533081, 1.1909772157669067, 0.4152063727378845, 1.0673387050628662, -0.17754937708377838, -0.1439955234527588, 0.824470579624176, -0.20710885524749756, -0.038572460412979126, 0.27911943197250366, 1.9070972204208374, -0.39433613419532776, 0.7251296043395996, -0.09780700504779816, -0.6094712615013123, 0.7099243402481079, -0.856526255607605, 1.5567537546157837, -0.11990942806005478], [0.6658657789230347, 2.036801815032959, 0.342941015958786, -1.1794228553771973, 0.9940159320831299, 1.7625811100006104, -0.0678764134645462, -0.3940020203590393, -1.1299551725387573, -0.4605576992034912, -1.386254072189331, 1.0377907752990723, -0.5008681416511536, -1.0791525840759277, -0.012997605837881565, 0.5407549738883972, 0.14176678657531738, -0.6851532459259033, -0.4269087612628937, -0.5959010124206543, -1.4231687784194946, -1.0790355205535889, -0.9049251079559326, 0.6466757655143738, -0.9833264946937561, 0.43764936923980713, 0.3155507743358612, -1.7870135307312012, 1.2866202592849731, 0.27463653683662415, -1.1470423936843872, -1.291176438331604, 0.9457209706306458, 0.8459200263023376, 0.9743867516517639, -2.5712947845458984, 0.8295427560806274, 1.4274775981903076, -1.1995558738708496, -0.23597903549671173, -1.2023513317108154, 0.8201639652252197, -0.9935081005096436, 0.8836703896522522, 1.0077701807022095, 1.0268616676330566, 0.18511056900024414, -1.23578679561615, -1.1148267984390259, 1.0803112983703613, 0.7565343379974365, -0.5385770201683044, 1.7482140064239502, -1.9557870626449585, -0.42810946702957153, -0.0008007276919670403, 0.2614201307296753, 0.4804191589355469, 1.3780909776687622, 0.8433747291564941, -2.678032875061035, 0.6677515506744385, -1.0760128498077393, 0.9666473269462585], [-0.8606531023979187, 1.0747672319412231, 0.5751677751541138, 0.15079393982887268, 0.16737833619117737, 0.9082800149917603, 0.6942921280860901, -1.4818214178085327, 0.9165138602256775, -0.6152479648590088, -0.9899073243141174, -0.6295152306556702, 2.0460195541381836, -0.1580977886915207, 0.19453419744968414, 0.046078577637672424, -1.7586461305618286, 1.3159898519515991, -0.4975410997867584, -2.3290438652038574, -0.3051280975341797, 0.9915889501571655, 1.6160694360733032, -0.39519524574279785, 1.2670148611068726, -0.4530748128890991, -0.15898475050926208, 0.8093795776367188, 0.05771531164646149, -0.9843148589134216, 0.10370750725269318, 0.4545605480670929, 0.8056816458702087, -1.8032498359680176, -0.8537614941596985, -0.5043407678604126, 0.16235408186912537, -0.8457948565483093, 1.0580053329467773, -0.405353307723999, -0.6650158166885376, -0.6689910888671875, -0.5164340734481812, 0.7714230418205261, -1.301045298576355, 0.43956997990608215, 0.6741496920585632, -1.1941874027252197, 0.8150593638420105, 0.34176480770111084, -1.1355741024017334, 1.2146320343017578, -0.6258150339126587, -1.600571870803833, 0.8890888094902039, -0.05961062014102936, 1.8286397457122803, 1.2919378280639648, -0.5065739750862122, -0.007173486985266209, 0.2646131217479706, -2.4300835132598877, -1.1484915018081665, -0.6831836104393005], [-1.2601792812347412, 1.2153480052947998, -0.40284210443496704, 0.22447125613689423, -0.8479255437850952, 0.005076170898973942, 0.050138041377067566, -0.2349199652671814, 2.2267520427703857, 1.6941461563110352, -1.1078392267227173, -2.245270252227783, -0.8985239863395691, 0.4877685606479645, -0.8559707999229431, -0.7395144104957581, -1.4215983152389526, 1.1350430250167847, 1.3680397272109985, 3.153759717941284, 0.7549452185630798, -0.7953699231147766, -0.8263064622879028, 0.40246427059173584, 0.47370028495788574, 0.04786150902509689, 0.15974439680576324, 0.6574006676673889, 0.3026118576526642, 1.0002422332763672, -1.183313012123108, 0.6458344459533691, 0.30507969856262207, -0.4895702004432678, -1.0598119497299194, -0.1008143424987793, 0.8230153918266296, 1.5965962409973145, -0.7761374115943909, 1.8702137470245361, 0.6536742448806763, 2.5738885402679443, 1.427031397819519, 0.9788144826889038, 0.6242195963859558, -1.1758313179016113, 0.3360106647014618, 1.6794244050979614, -1.6176962852478027, 0.07984323799610138, -0.8964236378669739, -0.14582259953022003, -2.282099723815918, -2.1357460021972656, -0.6368613839149475, -0.48495328426361084, 0.15514202415943146, -0.2741950452327728, -0.568466067314148, -0.635417103767395, -1.0836554765701294, 0.6021008491516113, -1.3394570350646973, -0.21907095611095428], [1.8506722450256348, 0.013115709647536278, -1.7970274686813354, -0.9916908144950867, 3.005366802215576, -0.6079307794570923, 1.538978934288025, -1.1389822959899902, 0.2370266616344452, -0.6834495067596436, -3.0026051998138428, -0.9922608733177185, 0.1526193916797638, -0.25651657581329346, -0.5311351418495178, 0.35924336314201355, 0.9597519040107727, -2.4322903156280518, -1.522969365119934, -1.6769732236862183, 0.09092656522989273, 0.5373790264129639, 1.152912974357605, -1.0974242687225342, -0.9949998259544373, -1.0278218984603882, 0.45403537154197693, -0.024670004844665527, 0.4300585687160492, -0.34595808386802673, -0.15591920912265778, 1.748347282409668, -0.12252064794301987, -1.459871768951416, -0.35001012682914734, -1.7386735677719116, -0.2140006572008133, 1.7994178533554077, 2.5473361015319824, 1.579811692237854, 1.030395269393921, 0.8634189963340759, -0.6126827597618103, -1.3516875505447388, -0.2032785266637802, 0.4965251684188843, -0.8030059337615967, 0.2135179191827774, -0.16760066151618958, 1.0259236097335815, -0.1983584463596344, -0.3327712416648865, 1.177500605583191, -0.5472663044929504, -1.3872630596160889, 0.11626709252595901, -1.0956000089645386, 1.041808009147644, -0.3676063120365143, -0.8816413879394531, -0.5180834531784058, -0.5853655338287354, 0.17606058716773987, 1.164458155632019], [-1.336718201637268, -0.6613689064979553, 0.40212786197662354, 0.16583620011806488, 1.202641487121582, -0.5686755776405334, -1.1514182090759277, -1.4302797317504883, 1.4569098949432373, -0.7643589377403259, -2.139655113220215, -0.5774332880973816, -0.31268906593322754, 1.4338419437408447, -0.8305178284645081, 0.06067494675517082, 0.5251401662826538, -0.3022609353065491, 0.08570907264947891, 1.151827335357666, -2.2082176208496094, -1.032190203666687, 0.24651946127414703, -0.17049285769462585, 1.326603651046753, 0.682924747467041, -0.3773898184299469, -0.7583708167076111, 0.818274736404419, 1.213265061378479, -1.3427584171295166, 0.382218599319458, 0.9259458780288696, 0.18478122353553772, 1.062415599822998, 0.7517086863517761, 1.228936791419983, 1.2866835594177246, -0.5608770847320557, -1.0513207912445068, -0.4229944050312042, -2.056534767150879, -0.10434667021036148, 0.0432632677257061, -0.9139348268508911, -2.1075665950775146, 0.05461496859788895, 0.49213334918022156, 0.4648006856441498, -1.6909775733947754, 1.1045364141464233, -0.28506532311439514, 0.0884377658367157, -0.9646721482276917, -0.45262983441352844, -0.28243201971054077, -1.1172415018081665, -1.304364800453186, 1.2913858890533447, -1.0492926836013794, -0.4238247573375702, -0.03478844463825226, 1.9217352867126465, -1.9710084199905396], [-1.174280047416687, 1.9140969514846802, 1.5361385345458984, 1.8865503072738647, -1.84092116355896, 0.11975281685590744, -0.6984719038009644, -0.05727490037679672, -0.7306919097900391, -0.43247804045677185, -1.6662886142730713, 1.2656525373458862, 1.3249906301498413, -0.5135566592216492, -0.5569681525230408, -0.2197074592113495, 0.042464956641197205, 0.281279057264328, -0.4676807224750519, 0.027392346411943436, -1.3382692337036133, -0.6717894077301025, -0.9595265984535217, 0.8292511701583862, 0.7302306294441223, 0.06091504171490669, -1.082169532775879, -0.18212604522705078, 0.7082582116127014, 1.6187363862991333, 0.658382773399353, 0.7950959205627441, 1.6639282703399658, -0.41596266627311707, 0.9776178598403931, 0.17509277164936066, -0.32665035128593445, -0.5926182270050049, 0.8095889091491699, -0.5229476690292358, 2.4619383811950684, -0.3292403221130371, 1.3137730360031128, -1.74606454372406, -1.0108245611190796, 0.7027367949485779, 0.04836318641901016, 0.29360660910606384, -0.9167594909667969, -0.6379808783531189, 0.2906687259674072, 0.9681942462921143, -0.8479950428009033, 0.7720344066619873, -0.4335669279098511, 0.755719006061554, -0.46462464332580566, -0.6769580841064453, -2.923339605331421, -0.27412736415863037, 0.8741111159324646, -0.17986980080604553, 0.35122251510620117, -0.5795970559120178], [0.6718090772628784, 0.0972880870103836, 0.415006160736084, -0.6356048583984375, 0.023063180968165398, 0.8605497479438782, -0.35720136761665344, 0.6842341423034668, -1.3242762088775635, -0.4852636158466339, 0.9742957949638367, -0.7269531488418579, -0.014042878523468971, -1.0005180835723877, -0.2991296052932739, 0.11406674981117249, -0.33530572056770325, -1.0768202543258667, 1.3319979906082153, -0.6411463022232056, -2.3603785037994385, -0.8141049146652222, 0.200271338224411, 0.3403467833995819, 0.3856259286403656, 0.9839684963226318, -0.7910593748092651, -0.6090916991233826, -0.5839974284172058, -0.35180771350860596, 0.8275722861289978, -0.34785106778144836, -0.8762142658233643, 0.5641381740570068, 0.7060225605964661, -1.2144893407821655, 1.0113978385925293, -0.2148267775774002, 0.5405350923538208, -1.3235610723495483, 0.24624767899513245, -0.16690964996814728, 1.1940422058105469, -1.6098130941390991, -0.6504095196723938, -0.2932431399822235, -1.9451732635498047, -0.6887162923812866, -0.2305462807416916, -2.4692270755767822, -0.5664952397346497, 0.1338917464017868, 0.3399493098258972, 0.6836364269256592, 0.46251246333122253, -1.279744029045105, -0.9213828444480896, 1.013674259185791, -0.008081506006419659, 1.4036579132080078, 0.1335466206073761, 1.1206556558609009, -0.7663206458091736, 0.481035441160202], [-0.8191959261894226, 0.34309542179107666, 0.8825365304946899, -0.5272786021232605, 0.14848999679088593, 0.9091359376907349, -0.5949144959449768, -0.5983490943908691, -0.5682294964790344, -1.1377290487289429, -0.49780914187431335, -0.1143530085682869, 1.0149943828582764, 0.26202234625816345, -0.5262070298194885, 0.7847333550453186, 0.008750866167247295, 0.6063926815986633, 2.1667251586914062, -0.4595973491668701, -0.09574491530656815, 0.3280380964279175, 1.2106547355651855, -2.050201654434204, 1.354548454284668, -0.22203421592712402, -0.4449491500854492, -1.2273496389389038, 0.8168683648109436, 0.7635215520858765, 1.231399655342102, -1.6628540754318237, -1.9440956115722656, -0.1537116914987564, -0.3919219970703125, -0.5991958379745483, -0.34954142570495605, 0.22464069724082947, -0.40911558270454407, -2.6235265731811523, -0.5769472718238831, 1.2993403673171997, -0.21041710674762726, -0.5515425801277161, 1.0019376277923584, 1.3982161283493042, -2.053492784500122, -0.09178704023361206, -0.42722129821777344, 0.664370596408844, 0.29736125469207764, 1.8037261962890625, -0.6873728036880493, 0.785962700843811, -0.09202220290899277, -1.8327949047088623, 0.41309455037117004, -1.2409149408340454, 0.4520058333873749, -0.5217958688735962, 0.49791377782821655, -1.8679239749908447, -0.02237730473279953, -0.24007798731327057], [0.3242233991622925, 2.8576629161834717, 0.9611460566520691, -0.012486507184803486, 0.7958724498748779, 1.575904369354248, 1.583626389503479, 1.4512368440628052, 2.289722442626953, -0.26742392778396606, 0.4736485779285431, -2.0759661197662354, 1.435890555381775, -0.6366692781448364, 0.6208668947219849, -0.07339365780353546, 0.42208755016326904, 1.9886565208435059, -0.6818928122520447, -0.23249123990535736, -1.362876296043396, 1.3673210144042969, -1.7304309606552124, -1.4565023183822632, -0.25321444869041443, 0.6165845394134521, -0.6914817094802856, 0.6952551603317261, 0.8295767903327942, 0.1904841810464859, -1.226002812385559, 0.48831748962402344, -0.5669021010398865, -3.213043451309204, -0.16408957540988922, -0.8257747292518616, -0.4900934100151062, -0.1955776810646057, -0.13009801506996155, -0.04419715702533722, 0.4267033338546753, -0.15252211689949036, -0.9450140595436096, 0.2845921814441681, 0.871438205242157, 1.282550573348999, -0.4618777334690094, -1.3655328750610352, -1.0982824563980103, -0.7818112969398499, 0.26024892926216125, 0.16382445394992828, 1.3814799785614014, 0.08403719961643219, 0.6563994288444519, -0.07075007259845734, -0.5589127540588379, 0.0017130946507677436, -0.08333922922611237, -0.3201524615287781, 0.26707810163497925, -0.20675398409366608, -0.5498117804527283, 0.11062860488891602], [-0.8472493290901184, -1.2862765789031982, -0.08782724291086197, 1.3809565305709839, 0.8893683552742004, -0.0850529670715332, -0.17562194168567657, 0.31356191635131836, -0.6968732476234436, 1.320364236831665, 1.3078205585479736, 0.18385303020477295, 1.5216399431228638, 1.07213294506073, -0.3808208107948303, 0.7020481824874878, -0.224257230758667, -2.1899471282958984, -0.8597103953361511, -0.8974117040634155, -1.726266622543335, -0.978847324848175, -0.3878189027309418, 1.3473620414733887, 1.006014347076416, -0.11633910983800888, -0.08383996039628983, -0.19182409346103668, 0.7652649879455566, 1.1531319618225098, -1.0170183181762695, -0.06414318084716797, -0.9756874442100525, 0.34444186091423035, 1.080227017402649, -0.9893010258674622, 0.007762709632515907, 0.15789766609668732, 0.661699652671814, 0.3217034935951233, -0.4667351245880127, -0.5898663997650146, 0.7695590257644653, -0.7693347334861755, -0.24530768394470215, 0.6624746918678284, 1.2478561401367188, 1.4366329908370972, 0.8940734267234802, -0.5743796825408936, 0.23741021752357483, -0.47264355421066284, 0.28330761194229126, -2.6601083278656006, 0.6107940077781677, 0.6583137512207031, 2.9248459339141846, 0.3252442479133606, -0.21992932260036469, 0.25109612941741943, 0.9850058555603027, 0.60549396276474, 0.2954319417476654, -0.3490193784236908], [-1.3326555490493774, -0.7273761034011841, -1.4940184354782104, 0.5205016732215881, -0.5270110964775085, 0.9956151247024536, 2.1841988563537598, 0.3851359486579895, -1.2250374555587769, -1.0814841985702515, 0.5836172699928284, 0.7631968259811401, -0.3823864459991455, 0.3344651758670807, 0.5764251947402954, 0.46189987659454346, -0.9331075549125671, 0.3745521306991577, 0.19855834543704987, -0.8400079607963562, -0.10140977054834366, 0.7375427484512329, 0.15073257684707642, -1.4552605152130127, 1.451263189315796, 0.8357791900634766, 1.0490461587905884, 1.4152289628982544, -0.7585562467575073, 0.29137566685676575, -1.6929436922073364, 1.1852643489837646, -0.7994329333305359, -1.1862975358963013, 0.8063615560531616, -1.5407413244247437, -0.4138800799846649, -2.2790000438690186, 1.8491419553756714, 2.5229387283325195, 1.0260779857635498, -0.5078701972961426, 0.6933580636978149, 0.48830851912498474, -0.10421445220708847, -0.9329935908317566, -1.1313689947128296, 0.8613265752792358, 0.0909704640507698, -0.5167424082756042, -0.8686397671699524, 1.893654227256775, -0.5214899778366089, 1.1669397354125977, -1.733689785003662, 1.0975862741470337, 0.07620256394147873, 0.4727121591567993, -0.8911106586456299, -0.424912691116333, -0.2004937082529068, -1.2694569826126099, -1.1138330698013306, -1.0753506422042847], [0.6309054493904114, -0.10144180804491043, -0.9424936175346375, -0.9327714443206787, 0.4808815121650696, 0.35325923562049866, 0.41836807131767273, -0.08659224957227707, -0.9456019401550293, -1.096785068511963, -0.9686146974563599, 0.1234624981880188, 1.1025422811508179, 0.24570520222187042, 1.0538699626922607, 1.0701630115509033, 0.7948281168937683, -1.291552186012268, -0.737108051776886, -0.6799885034561157, 1.5594943761825562, -0.7746987342834473, 0.5039177536964417, 0.8283807039260864, -0.009160705842077732, -0.7139724493026733, 0.7299562692642212, -2.118088960647583, -0.9042849540710449, -1.13009512424469, -1.2390437126159668, -0.29199478030204773, 0.3184596300125122, -0.8448649644851685, 0.3180556893348694, -0.4226265847682953, -1.4800989627838135, 1.9874809980392456, 0.9329040050506592, -0.8128795623779297, -0.737667441368103, 0.17909684777259827, -0.270070344209671, -0.39204224944114685, 0.8059601783752441, 1.3501275777816772, -1.4235082864761353, -1.2457152605056763, -0.698023796081543, 0.8715112209320068, -1.2639590501785278, 1.139299988746643, 0.8883201479911804, 0.4699154496192932, 1.141789436340332, -1.5660983324050903, 0.5254888534545898, -0.4794059991836548, 0.88517826795578, -0.5403012633323669, -1.4669766426086426, -0.7928329110145569, 1.6036700010299683, 0.22504264116287231], [0.21807536482810974, 1.034673810005188, -0.27112865447998047, -1.546744465827942, -2.267291307449341, -0.18342836201190948, 0.35132575035095215, -0.6001712679862976, -0.37770453095436096, -0.3890498876571655, 0.674189567565918, -0.6127959489822388, 0.7521865367889404, 0.096961110830307, -0.21176549792289734, 1.3995434045791626, -0.019645294174551964, 0.1643563210964203, -0.2641673982143402, 0.3322364091873169, -1.0856560468673706, -0.7216845154762268, 0.7853544354438782, -0.44534146785736084, 0.4561787545681, 0.3308973014354706, -0.09047617763280869, -0.26610490679740906, 1.13493812084198, 1.9525129795074463, -0.49664434790611267, 0.48347094655036926, -1.2884316444396973, 0.061127517372369766, 0.7285657525062561, -0.8668779730796814, -0.30594104528427124, -0.40287861227989197, -1.3073182106018066, -1.1538926362991333, 0.2116934061050415, -0.2742445170879364, -0.9878891110420227, -1.201345443725586, -1.4324839115142822, -0.41008397936820984, -0.016601402312517166, 0.3147962689399719, 0.9183701276779175, 0.06496145576238632, -1.5371941328048706, 0.025670776143670082, -0.5064172148704529, -1.2505265474319458, 0.7268591523170471, 0.6217544674873352, 0.329845130443573, -0.6706042289733887, -0.36429962515830994, 0.2764151692390442, 0.6964346170425415, 0.0959421843290329, -0.37014973163604736, 0.09199471026659012], [0.7381665706634521, -0.440753310918808, -0.961662232875824, 1.683472990989685, -1.6883167028427124, -1.8044668436050415, 1.297235369682312, 0.3304690420627594, 0.7807584404945374, 0.5534898042678833, 1.6164995431900024, -1.0096796751022339, -1.0155457258224487, -0.520663857460022, 1.5801067352294922, 0.9028440713882446, 0.7808574438095093, -0.16870668530464172, -1.3162349462509155, -2.0840516090393066, 0.8370333313941956, 0.3556906282901764, -0.16315142810344696, 0.663378119468689, 0.7556813359260559, 0.5422397255897522, -2.0770933628082275, 1.1161655187606812, -0.5409088134765625, -0.14120803773403168, 0.6929000020027161, -0.6499976515769958, 1.3427892923355103, -0.23836787045001984, 0.20046323537826538, 0.7423864006996155, 0.7592893242835999, -0.13810333609580994, 0.21306996047496796, -0.8224732875823975, -1.5291939973831177, -1.1037368774414062, -0.5291832089424133, -0.09661480039358139, 0.2477758526802063, -2.5500292778015137, -1.4200749397277832, -0.5753350853919983, 0.41115516424179077, 0.11277634650468826, 1.9269405603408813, 0.8957124948501587, -0.9245930910110474, -0.019650960341095924, -0.8699221014976501, 0.861283540725708, -1.4570335149765015, 0.5411601066589355, -0.6631753444671631, -0.19176863133907318, 0.06952414661645889, -0.556646466255188, -0.48087263107299805, -0.38629278540611267], [-0.18159589171409607, 0.6232479810714722, 0.7543498277664185, 0.18962031602859497, 0.747083842754364, 1.467429757118225, -0.8418270349502563, 0.2965952455997467, 0.40577375888824463, -1.8228217363357544, 0.1274651288986206, -0.773623526096344, 0.7938412427902222, -0.6520751118659973, -0.07956336438655853, -0.6166653633117676, -0.3862449824810028, -1.32981538772583, 1.0809046030044556, -1.0363954305648804, -0.4381411671638489, -0.15041819214820862, 0.3935662806034088, 0.6046974658966064, -0.5668630599975586, 0.19894783198833466, 1.7272344827651978, 0.41774868965148926, -0.937505841255188, -0.49619489908218384, 1.5241073369979858, 1.1814881563186646, 0.8023000359535217, 2.5961506366729736, 0.18185071647167206, -0.4087394177913666, -1.3024632930755615, -1.2754062414169312, -1.799890398979187, 2.1244516372680664, 0.4112926125526428, 1.7297563552856445, -2.2089102268218994, 0.8876376748085022, -0.5564510822296143, 0.9745684266090393, -0.6477425694465637, 0.22333167493343353, 0.7303387522697449, 0.2987820506095886, 1.370208978652954, 0.5897871851921082, -1.0479762554168701, -0.7711161971092224, -1.5288430452346802, -1.2604914903640747, -0.7066977620124817, -1.568847417831421, -0.3217617869377136, 0.06760153919458389, 0.7073805928230286, 0.6109667420387268, 0.9498923420906067, -0.19351688027381897], [-0.49152085185050964, 0.036245472729206085, -1.1049860715866089, 1.3335930109024048, 0.49332156777381897, 2.4480388164520264, -1.7411813735961914, -1.0949265956878662, 0.5481184124946594, -0.19407804310321808, -1.7150237560272217, 1.8314592838287354, 0.917142391204834, 0.19552968442440033, 0.9140870571136475, 1.2615388631820679, -0.2960016429424286, 0.4982365667819977, 0.3128780722618103, 1.314251184463501, 0.4651775360107422, 0.8521171808242798, 0.5252605676651001, -1.1096422672271729, -0.18090566992759705, -0.1706068217754364, -0.15005359053611755, -0.3332882821559906, -0.6175182461738586, -0.028253812342882156, -0.46506598591804504, -0.043304868042469025, 0.8497086763381958, -0.47660261392593384, 1.1130512952804565, 1.8845505714416504, -0.11561821401119232, 0.47449108958244324, 0.09881675243377686, -2.093733549118042, -0.12539725005626678, -1.3138922452926636, 1.595245122909546, -0.0564611442387104, 1.694580078125, -0.784424364566803, -1.5466989278793335, 0.8184369206428528, -1.0634024143218994, -0.4563310742378235, 0.6939913630485535, -0.7881401777267456, 1.4416351318359375, 1.5501832962036133, -0.4633176922798157, -0.6144096255302429, 0.3899451792240143, -0.0275775995105505, -0.2058323472738266, -0.3718257546424866, -0.392203152179718, 0.1646115630865097, -0.22956620156764984, -0.22005106508731842], [-0.18962755799293518, 1.4053231477737427, 0.05849615857005119, -1.4443029165267944, 1.093026041984558, -1.5823558568954468, -1.8201626539230347, 1.0121492147445679, -0.12399277836084366, 0.31184375286102295, 0.6876484751701355, 0.004519931506365538, 1.9359413385391235, -1.3787670135498047, -0.056078486144542694, -1.1481748819351196, -1.2296279668807983, 0.2488427460193634, -0.632416844367981, -0.6521012187004089, -1.1115256547927856, -1.0969666242599487, -0.20695152878761292, -0.6429594159126282, -0.9842621088027954, 0.29227104783058167, -1.868325114250183, 0.6920049786567688, 0.19900503754615784, -0.9086755514144897, 0.7564969658851624, 1.5700045824050903, 3.2398574352264404, 0.7351970672607422, -0.4773764908313751, -1.1810864210128784, -2.0763068199157715, -0.508857786655426, 1.1662043333053589, 0.8439779281616211, 0.26637697219848633, -2.304600954055786, 1.4031387567520142, -1.6824499368667603, 0.1943311095237732, 0.9747512936592102, -0.08570536971092224, -0.5253961086273193, -1.14815092086792, -0.09058572351932526, 0.5894501209259033, 0.5970019698143005, 1.0161226987838745, 0.37394511699676514, 1.2558698654174805, -1.0699269771575928, -0.0870170071721077, -2.0380403995513916, 2.7530806064605713, 1.1404224634170532, 0.03349503502249718, -0.5818687677383423, -0.38439086079597473, -0.3760254383087158], [1.351545810699463, -0.6302489042282104, 0.5179042816162109, -1.41119384765625, -0.9864246249198914, 0.26045235991477966, -2.0152645111083984, -1.838512659072876, 0.2569999694824219, -0.05864716321229935, 0.5167398452758789, -0.4072943329811096, 0.5589330792427063, -1.0077975988388062, -0.8242683410644531, 0.5573317408561707, -0.7945144176483154, 0.4345415234565735, 1.858884334564209, 0.40338578820228577, -0.8976978659629822, -0.8471246957778931, 0.2075064331293106, -0.11239518225193024, 0.32375726103782654, 0.1628577709197998, 0.9957496523857117, 1.551896333694458, 1.4495142698287964, 1.0738624334335327, -1.1997621059417725, -0.6706965565681458, 0.2377478927373886, -0.04682210087776184, -0.3096896708011627, 0.3110414743423462, -0.4200987219810486, 1.4501651525497437, -0.8234866261482239, 1.0236772298812866, 1.4813435077667236, -0.07922157645225525, -0.5545201301574707, 1.1945607662200928, 0.3144610822200775, 0.526391863822937, 2.0014686584472656, 2.216111660003662, 1.4534409046173096, 0.11228063702583313, 0.5604839324951172, 0.8110145926475525, 1.5374301671981812, 0.8026045560836792, -1.6133028268814087, 0.4641244411468506, 1.1476904153823853, 1.0486574172973633, -0.7252228260040283, -0.3968222141265869, 0.1022283211350441, -1.337032675743103, 0.19783101975917816, 0.10694502294063568], [-0.05887843295931816, -1.6830272674560547, 0.526739776134491, 1.3045628070831299, -0.8679972290992737, -1.228536605834961, -0.27394238114356995, -1.0758384466171265, -0.5029253363609314, 0.30709144473075867, -0.23646193742752075, -0.03543310612440109, 0.014313377439975739, 0.4423326253890991, 1.2277462482452393, 0.2444080412387848, -0.1834668517112732, 1.1618226766586304, -0.12491589039564133, 0.025168130174279213, -0.38016802072525024, -0.21187645196914673, -0.1918039321899414, -0.18772639334201813, 0.17633883655071259, 0.616200864315033, 0.10890527069568634, -1.9491877555847168, 0.03289934992790222, 0.5428162813186646, -0.4930962026119232, -0.22827506065368652, -0.4170151352882385, 0.39727285504341125, 0.39199313521385193, 1.2340283393859863, 0.5871832370758057, -0.12861749529838562, -0.7881706953048706, -3.617387294769287, 0.27122920751571655, -1.2980308532714844, -0.18720649182796478, 1.925366997718811, -0.04314194619655609, 1.0275572538375854, 0.11231962591409683, 0.6763219833374023, -0.5247469544410706, -0.821913480758667, 2.4717719554901123, 0.9258788228034973, 1.5979218482971191, -0.44697442650794983, -0.4813072085380554, 0.6037863492965698, 0.3342694938182831, 0.6596133708953857, -0.521774172782898, 0.06563031673431396, -0.43896484375, 1.2297836542129517, 1.0655958652496338, -2.188206672668457], [-0.714945912361145, 0.0720834732055664, 0.6825889348983765, 0.2676661014556885, -1.3978639841079712, -1.7525039911270142, 0.727914035320282, -0.23536352813243866, -0.9669831395149231, -0.3597525954246521, 0.5786861181259155, 0.18679435551166534, -0.1679442673921585, -0.2648420035839081, -0.06484653800725937, -0.7882022261619568, 0.22438329458236694, -0.7910192608833313, -2.1596810817718506, -0.6596878170967102, 0.42333051562309265, -1.7497069835662842, 0.136344775557518, 1.3717310428619385, 0.6917685866355896, -0.7213305830955505, 0.5739601254463196, -0.4038779139518738, 0.9012284278869629, 1.8843986988067627, -1.5452076196670532, 0.8108687996864319, 0.5421981811523438, 0.5207122564315796, -0.29078271985054016, -0.13649077713489532, 1.2930136919021606, 1.9063977003097534, -1.1068769693374634, 0.22492700815200806, 0.8251208662986755, 0.42185941338539124, 0.4453803300857544, 0.07503965497016907, -0.05427752807736397, -1.0942325592041016, 1.513369083404541, -1.1100119352340698, -1.8045300245285034, -0.5218731164932251, -0.638654351234436, -0.20759695768356323, 0.22159475088119507, 1.2589447498321533, -0.8832062482833862, -0.1873667687177658, 0.11528511345386505, -1.393264889717102, -0.9072989821434021, -0.06750183552503586, 1.0838907957077026, -0.3395751118659973, -0.9711999893188477, 1.0593369007110596], [-2.303457498550415, 1.3440773487091064, 0.3577430546283722, -0.3213289678096771, -0.9422869086265564, 2.262526035308838, 0.5906925201416016, 0.6221092343330383, 1.443423867225647, -1.0315648317337036, -1.7493925094604492, -0.29955577850341797, -0.4063504636287689, 2.043372631072998, 0.2389683574438095, -0.28657957911491394, 1.7859975099563599, -1.3213169574737549, -0.9485487937927246, -0.7305604219436646, -1.1533682346343994, 0.1350610852241516, -0.5896556973457336, 1.4264228343963623, -0.2674098014831543, -0.7739099264144897, -1.2012766599655151, -0.793885350227356, -1.1600780487060547, -0.8313583731651306, -1.3488000631332397, 0.3202272951602936, 0.34765300154685974, 0.2307795286178589, -0.5173925757408142, -0.1141490787267685, 0.8968040943145752, -0.3836418688297272, 0.11289054155349731, -0.048172060400247574, -0.0929899662733078, 0.2957342565059662, -0.7277225852012634, -0.9913939833641052, -0.9919152855873108, -1.3526544570922852, -0.35256585478782654, 0.20774684846401215, -1.2030458450317383, -0.05732232704758644, -1.7743090391159058, -0.2268940508365631, -0.4837436079978943, 2.194620132446289, -0.07725337147712708, -1.043250560760498, 0.41639041900634766, 0.5788331627845764, -0.6715959906578064, 1.72182035446167, -1.246976375579834, -0.9782356023788452, -0.8388015627861023, 1.3538936376571655], [1.3740060329437256, -0.5245395302772522, 0.07351578772068024, 0.032991405576467514, -0.20957686007022858, -0.7174263000488281, -0.19703343510627747, 0.10501531511545181, -1.4858806133270264, 1.0918174982070923, 0.8576308488845825, -0.871183454990387, 0.5173776745796204, 1.319169282913208, 0.678300142288208, 0.48272204399108887, -1.1435322761535645, 0.6162102222442627, 1.5401959419250488, -1.9336037635803223, 0.1897725760936737, -0.2134491205215454, -0.7241358160972595, 0.6398645639419556, -0.5948051810264587, 1.261612057685852, 1.254349946975708, -0.7195889353752136, 1.4785740375518799, -0.14238087832927704, -0.5424322485923767, -1.5828728675842285, 0.5889236927032471, -0.1763879358768463, -1.4181373119354248, -0.7418984770774841, -1.1765661239624023, 0.20571239292621613, -1.8683733940124512, -0.37619227170944214, 2.0665128231048584, 0.7238897085189819, 0.1861937940120697, -1.3403664827346802, -0.32072216272354126, -0.2260081171989441, -0.31824082136154175, 1.4124319553375244, 0.34387609362602234, 0.4646075665950775, -0.7830412983894348, -0.8281449675559998, -0.2127222716808319, -1.029979944229126, -1.0403209924697876, 0.0453401654958725, -1.7585982084274292, 0.523770809173584, 0.0326642170548439, -0.5692111253738403, 0.9881008267402649, 0.5629479885101318, -0.33880722522735596, -0.5209763646125793], [0.770556628704071, 1.7236379384994507, -0.9896518588066101, 0.9902275204658508, 0.007048777770251036, -0.02411292865872383, 1.5537004470825195, -0.15967105329036713, -1.079533338546753, -0.753660261631012, -0.37319818139076233, 1.8777475357055664, -0.5647546052932739, 1.6288166046142578, 0.007782837841659784, -1.6521962881088257, -0.3370421826839447, 0.9538112282752991, -1.2164559364318848, 0.11229376494884491, -0.31890109181404114, -1.011301040649414, 0.17485980689525604, 0.5963749289512634, 1.864890694618225, 0.4076152741909027, 0.9619709849357605, 1.6355311870574951, 0.03804001584649086, 0.4887087345123291, 0.16679371893405914, 0.07834223657846451, -0.935366690158844, -0.7840150594711304, 1.052687406539917, -1.9147461652755737, 0.5768793821334839, -0.11958231031894684, 1.468553066253662, 0.7577335238456726, -0.9657408595085144, -0.16422852873802185, 0.20020586252212524, -0.5387738943099976, -1.5044935941696167, 1.6437848806381226, 0.7458634376525879, -1.2216485738754272, 0.1754896491765976, 0.5669531226158142, -1.4448773860931396, 1.6469441652297974, -0.857674241065979, -1.2669928073883057, -0.2714657485485077, -0.45673203468322754, -0.39669671654701233, 0.19226083159446716, -0.7527214884757996, -0.16523370146751404, 0.5233157873153687, 0.2940595746040344, -0.32210609316825867, -0.7747213840484619], [-1.0077145099639893, 1.3715057373046875, 0.21969448029994965, -0.757430374622345, 0.7930744886398315, 0.1520727127790451, -1.049241304397583, -0.7505577802658081, -0.18368254601955414, 0.7917050123214722, 1.6722159385681152, 0.9834105968475342, -0.5811762809753418, -1.513169527053833, 0.7399059534072876, -0.07121817022562027, 0.5465868711471558, 0.11397664248943329, -1.285573959350586, 0.1291157603263855, -0.3189244568347931, 1.6505693197250366, -0.38131245970726013, -2.1247124671936035, -0.726045548915863, -0.5840892195701599, 0.9712029099464417, -1.9254816770553589, -0.14798586070537567, -0.6990912556648254, -0.6721115112304688, 0.6918473839759827, -2.1271653175354004, -0.03887222334742546, -1.46125066280365, -1.164393424987793, -0.0812525600194931, -0.30553337931632996, 0.1349560171365738, -0.533467710018158, 1.9248063564300537, -0.47010788321495056, 0.820398211479187, -0.30240288376808167, 0.48279422521591187, 0.2487601488828659, 1.031815528869629, -0.3037285804748535, -0.08435027301311493, -0.6935011744499207, 0.2695605754852295, 0.4946092367172241, 0.09122876822948456, -0.39400866627693176, -0.046161867678165436, -0.5909029841423035, -0.835883617401123, -0.16708241403102875, 1.0403119325637817, -0.7084073424339294, 0.10021958500146866, 0.6099466681480408, -0.35827940702438354, -0.4018993377685547], [0.2931307852268219, -1.0234586000442505, 0.022464299574494362, 0.8104127049446106, -0.47969111800193787, -0.841400682926178, 1.3297672271728516, 1.8185696601867676, 0.9517743587493896, 1.7784829139709473, -2.3950817584991455, -0.1349865198135376, -0.721397340297699, 0.25569361448287964, 1.833199381828308, -0.4706549644470215, 0.33752137422561646, 1.0190908908843994, 0.012824214063584805, -0.5779613852500916, 0.31452110409736633, -0.08548677712678909, -0.2864552438259125, -1.269241213798523, -0.35177358984947205, -2.1961050033569336, 0.6472300291061401, 0.6243272423744202, -0.06561341881752014, 0.15358661115169525, -3.1011836528778076, 1.4490662813186646, -2.364887237548828, -1.4864873886108398, 0.08477664738893509, -0.06598901003599167, -2.0739474296569824, 1.674471378326416, 1.2670687437057495, -2.2191057205200195, -0.17181496322155, -0.1726989448070526, 0.42978304624557495, 0.4241587817668915, 0.12200306355953217, -0.23542380332946777, 0.08089304715394974, -0.720802366733551, 0.14272429049015045, 0.05502020940184593, -0.7214915156364441, 1.421994924545288, -1.7172988653182983, 1.3225603103637695, 0.5539754033088684, 0.062188711017370224, -1.167039394378662, 0.6983781456947327, -0.23596876859664917, 0.4033481478691101, -0.06502393633127213, 1.5963689088821411, 0.286761999130249, -0.6658011078834534], [-2.4175760746002197, -0.5957497358322144, 0.6028379201889038, 0.5613420009613037, 0.39526763558387756, 0.03805744647979736, 0.7231016755104065, 1.0470761060714722, 1.1173210144042969, 1.0044682025909424, -0.25085389614105225, 0.5058062672615051, 0.6975590586662292, -0.3048182427883148, 0.6860743761062622, 0.2002810686826706, -0.2128472924232483, -0.6960904598236084, 0.0366070456802845, -0.5824602246284485, 1.335771918296814, -0.20410992205142975, -0.4719424843788147, -0.03470131382346153, -0.9372787475585938, -0.1296912431716919, -0.9401450157165527, 0.44770199060440063, 1.500241994857788, 0.1759500950574875, -1.10796320438385, 0.4113883078098297, 1.1981004476547241, 0.3733484745025635, 1.3930758237838745, -1.1508795022964478, -0.8873986005783081, -0.737862229347229, -0.25122594833374023, -0.15513744950294495, -1.3505727052688599, 0.7530462145805359, 0.405732125043869, -0.2625615894794464, 0.2952021062374115, 0.412397176027298, -1.5304462909698486, -0.5876151919364929, 0.3056183159351349, 0.8792082667350769, -0.9685689806938171, 0.7803810834884644, -0.03892146795988083, 0.6113944053649902, -0.07850779592990875, 0.777001678943634, 0.2609036862850189, 0.8398106098175049, 1.0601614713668823, 0.5956853628158569, -0.4310586452484131, 0.18828855454921722, 1.9506555795669556, 0.23327316343784332], [-1.7672741413116455, -0.8643211126327515, -0.8247837424278259, -1.189530849456787, 0.23530693352222443, -0.3191748857498169, -1.1682218313217163, -1.212298035621643, -1.6095067262649536, -0.17361824214458466, -0.10690584778785706, 0.4370848536491394, 0.6982705593109131, 0.3954932689666748, -0.6702155470848083, -1.0904991626739502, -0.2717520296573639, 0.26927870512008667, -1.5833362340927124, -0.11527080088853836, 0.5989442467689514, -0.020283866673707962, -0.699356734752655, -0.0260967705398798, -0.30169907212257385, -1.1036407947540283, 1.045511245727539, 0.5855796933174133, 0.016459129750728607, -0.14198777079582214, -1.2145957946777344, 0.19450141489505768, 0.6239405870437622, -0.5684189796447754, -0.864258885383606, -0.44215595722198486, 2.7958099842071533, -0.21250078082084656, 1.1821746826171875, 0.34508293867111206, -0.7104402184486389, -0.15384697914123535, -0.6853764057159424, 1.4622266292572021, -0.6067100763320923, 0.6045346856117249, 0.3566412925720215, 0.666335940361023, 0.3098335266113281, -1.4567934274673462, -1.1398749351501465, -1.0401866436004639, 0.8173472285270691, -0.8089367151260376, 0.36300238966941833, 0.5189744234085083, 0.6973119378089905, -0.7033752202987671, -1.7158502340316772, 3.0413057804107666, 3.077634572982788, -0.5684071779251099, 0.5279405117034912, -1.3747258186340332], [1.2944059371948242, 0.19656182825565338, 0.3644546866416931, 1.4383574724197388, 1.1117491722106934, 0.19506941735744476, 0.09345605224370956, 0.718071460723877, -0.8732951879501343, -0.05266103893518448, -0.8915729522705078, 1.110028862953186, 2.338104248046875, -0.1305508017539978, 0.22832657396793365, -0.5799669027328491, 0.030626138672232628, 2.1245813369750977, 0.6792778372764587, -0.5867804884910583, -0.31654858589172363, 0.039621371775865555, -1.6339927911758423, -0.32350361347198486, 0.8796833753585815, 0.6119994521141052, -1.4170645475387573, 1.4741601943969727, -1.6722798347473145, -1.581460952758789, 0.5638329386711121, -0.9945803284645081, -0.07539588958024979, 0.4470537006855011, -0.04602259397506714, -1.8142720460891724, 0.15538866817951202, -0.9816774725914001, -0.18518975377082825, 0.20970122516155243, 0.5756135582923889, 0.5842741131782532, -0.7302780151367188, 0.38674211502075195, -0.21102091670036316, -0.8204829096794128, -0.28809675574302673, 0.6384959816932678, 0.557863712310791, -0.7613934278488159, 0.577706515789032, -0.02966507524251938, 0.13370995223522186, 0.7169339656829834, 0.4798464775085449, -1.3009101152420044, -0.16871385276317596, -0.29150837659835815, -0.7727640867233276, -0.49936643242836, 0.6779095530509949, -0.06349195539951324, 0.22217631340026855, -0.825387716293335], [0.04895208403468132, 1.1842927932739258, 1.0158674716949463, 0.05858903378248215, 1.1100047826766968, -0.8822021484375, 0.015581324696540833, 0.7482654452323914, -1.160377025604248, -0.3295796513557434, 0.06454586982727051, 0.04560520872473717, -1.5994235277175903, -0.1734752207994461, 0.1913519650697708, -0.8991654515266418, 0.7514253258705139, -1.0048893690109253, -0.8812468647956848, 1.9586228132247925, 0.08107712864875793, 0.5720473527908325, -0.24815359711647034, 0.6290998458862305, -0.5338999629020691, -1.077928900718689, -0.7153191566467285, -1.0856411457061768, -0.21210291981697083, -1.9653335809707642, 0.2043924629688263, -0.6780532598495483, 0.024277430027723312, 1.9060178995132446, -1.165387511253357, 0.28840699791908264, 0.6057165265083313, -0.05768606811761856, -0.9398590922355652, -0.8121572136878967, 0.028291497379541397, 0.9749162197113037, -0.1858663558959961, 1.0908114910125732, 0.5616128444671631, -0.08979294449090958, -0.8173977732658386, -0.4801444113254547, -2.7935800552368164, 0.626972496509552, -0.30540168285369873, -0.31215280294418335, 0.6424647569656372, 0.8567492961883545, 0.26619210839271545, 0.024351684376597404, -0.23060312867164612, 0.24086058139801025, -1.1011985540390015, 0.7213616371154785, 1.5339808464050293, 0.9051164984703064, 0.3860990107059479, -1.2524759769439697], [1.4593013525009155, -0.9355592131614685, 1.1525136232376099, 1.5689878463745117, -0.7570788264274597, -1.3214187622070312, -0.10609849542379379, 2.7524938583374023, 0.09957224130630493, 1.5581837892532349, 1.312389850616455, 1.2463864088058472, -0.6316433548927307, 1.518502950668335, -0.48080241680145264, 0.4190370440483093, -0.6425341367721558, 0.18360283970832825, -0.2210550457239151, -0.05350516736507416, 0.7110717296600342, -1.3012638092041016, 0.08099447935819626, -0.8475316762924194, 0.8311114311218262, -0.0426115021109581, -0.4144410490989685, -0.4770505726337433, 0.367942750453949, -2.275733232498169, -0.5026167631149292, -0.6433461308479309, -1.2987138032913208, -0.5700703859329224, 0.11121074855327606, -0.27307549118995667, 1.2339566946029663, -1.3784780502319336, 1.2816779613494873, -0.29074129462242126, 0.991610586643219, -0.544084906578064, -0.6320759654045105, -0.4342988133430481, -0.9471037983894348, -0.8588390946388245, 0.022183014079928398, -0.6165582537651062, 1.4913580417633057, 0.18798403441905975, 1.7751238346099854, 1.436360478401184, 0.5666991472244263, 1.5965861082077026, 0.32460811734199524, 0.9307617545127869, -0.8970751166343689, 1.0316073894500732, 1.183870553970337, 1.140864372253418, -0.9528834819793701, -0.3206752836704254, -0.9063149690628052, -0.5417190790176392], [-1.1005116701126099, 0.6205775737762451, 1.7775717973709106, 0.8450959920883179, 0.7448096871376038, 0.5677599310874939, 1.3842496871948242, -1.0504056215286255, -0.06982135772705078, -1.6015028953552246, -1.547343134880066, -0.3852899968624115, -0.09287109971046448, 0.23224405944347382, 0.10320480912923813, -0.13207942247390747, -0.9379761815071106, 0.83756422996521, -0.1726658195257187, 1.017961859703064, 0.019924143329262733, 0.32520103454589844, 0.41736310720443726, -0.5885652303695679, 0.10192395746707916, -0.46209490299224854, 1.0043028593063354, 0.11147037148475647, 1.4750226736068726, -1.5320191383361816, -0.2255294919013977, 0.5284367203712463, -1.3645859956741333, 0.5876957774162292, 1.5351240634918213, 1.2432159185409546, -1.5496485233306885, 0.981721818447113, 0.6755186319351196, -1.6589136123657227, -0.6073135733604431, -0.22203364968299866, 0.0470610149204731, -0.2569647431373596, 0.7255361676216125, 1.2566726207733154, -0.3362812399864197, -0.8412402272224426, 0.3758904039859772, 0.6904046535491943, -1.107208013534546, -0.051596514880657196, -1.2546883821487427, 0.6840368509292603, -0.043160855770111084, 1.1328119039535522, 1.9077924489974976, -0.1714933067560196, 0.34677618741989136, 0.6895161271095276, 0.5799760818481445, 0.482413113117218, 0.9801653027534485, -0.7453687191009521], [1.098751187324524, 0.3160540461540222, 0.3147042691707611, -0.8308176398277283, -0.952226459980011, 2.2104785442352295, -0.21849453449249268, 0.42304399609565735, -0.945293128490448, -0.0015524579212069511, 0.26326343417167664, -0.5501789450645447, 0.9795194268226624, 0.4609432518482208, 1.5165941715240479, 0.17906798422336578, -0.5455908179283142, 0.3365444839000702, 0.42698732018470764, -0.6363381147384644, 0.5358356237411499, 1.9783991575241089, 1.501053810119629, -0.09048036485910416, 0.0819433256983757, -0.8126391172409058, -0.265925794839859, 0.0998602882027626, 0.1098540872335434, -2.1662070751190186, -0.3488146960735321, -1.8190630674362183, 0.973332405090332, -0.9685099720954895, 0.48294782638549805, -0.7727969288825989, 0.5447918176651001, 0.7312750816345215, -2.7572951316833496, -1.0697975158691406, -0.1642741560935974, -0.19286786019802094, 1.6852678060531616, 0.3495509922504425, 0.8403935432434082, -0.3629450798034668, -0.4122549295425415, -0.5822972655296326, 2.3339953422546387, 0.2747824788093567, -1.6510124206542969, -0.3869518041610718, -0.42711275815963745, -2.032615900039673, -1.8653912544250488, 0.04236694425344467, -0.7940273284912109, 0.9255543351173401, -0.2857827842235565, 0.4811403751373291, 0.19208672642707825, -1.0616209506988525, 0.7016831636428833, 0.6735796332359314], [0.9005953073501587, 1.6854413747787476, 0.24220828711986542, 0.1575329303741455, -0.02087073028087616, 0.30645325779914856, -0.3607155680656433, 0.28897958993911743, -1.096368670463562, -0.7475345730781555, -0.7381695508956909, -0.2688458561897278, -0.7796346545219421, 1.026673436164856, 0.48476511240005493, -1.9573912620544434, 1.0809348821640015, 1.3901550769805908, 0.7226812243461609, -0.010304886847734451, -2.3062965869903564, -0.3135601878166199, 0.9643550515174866, -2.0220561027526855, -0.4374924898147583, 0.023623919114470482, 0.22852283716201782, -1.9421149492263794, -0.28669944405555725, -1.6503779888153076, 0.04140700399875641, 1.7240970134735107, 1.3399020433425903, -0.39531320333480835, -1.1667544841766357, -0.08750206232070923, -0.11618391424417496, -0.589631974697113, -0.3962831199169159, 1.0051389932632446, -1.6664754152297974, -0.0321095734834671, 1.7482786178588867, 0.7792437076568604, -0.13195499777793884, -2.0723700523376465, 0.46873000264167786, 0.23486605286598206, 0.3160681426525116, -0.8774384260177612, 1.7302579879760742, 0.1709667146205902, 0.1269349455833435, -0.023748518899083138, -0.27631527185440063, -3.0736069679260254, 0.18287676572799683, 1.0802069902420044, -0.13423235714435577, -0.10939253866672516, 0.555992603302002, 0.5285136699676514, 0.4186471402645111, 1.9331927299499512], [-1.4945220947265625, 0.2624574601650238, 0.952999472618103, 0.6336147785186768, 0.7869037985801697, -2.725895404815674, 0.828336775302887, -0.5547200441360474, -0.27901211380958557, 0.06517236679792404, -0.12304054945707321, 0.5521885752677917, 1.6157432794570923, -0.34838050603866577, -0.8079948425292969, -0.11215682327747345, 0.8762845396995544, -0.2980117201805115, 0.04564126580953598, -0.4637439250946045, 0.3866170644760132, 0.28194889426231384, 0.087895467877388, 0.7109459042549133, 0.6938967704772949, -0.14134572446346283, -0.384687602519989, 0.8584821224212646, -0.938356876373291, -0.7813348770141602, 0.06584212929010391, 0.6458232998847961, -0.3420444428920746, 1.7480697631835938, 1.1967896223068237, 0.6459971070289612, -0.5487722158432007, -0.7377521395683289, 0.4289230406284332, 0.5636457800865173, -0.9398093819618225, -0.18546241521835327, 0.47495609521865845, 1.420473337173462, 0.5971762537956238, -0.06949906051158905, -0.28841522336006165, 1.1993310451507568, 1.6594536304473877, -1.1248300075531006, 0.41512104868888855, -0.12600183486938477, 0.32830944657325745, -1.0548062324523926, -0.1812756359577179, -1.2718511819839478, -1.6822946071624756, 1.1582311391830444, 1.8498203754425049, 0.9402034878730774, -1.0943377017974854, 0.8958924412727356, 0.7515637278556824, 1.9698829650878906], [1.0034801959991455, 0.6764501929283142, 1.7794125080108643, -0.3884594142436981, -0.23803706467151642, -0.8059816956520081, -0.11724083870649338, 1.1618614196777344, 0.4539409577846527, 0.029202032834291458, -1.1983611583709717, -0.39367300271987915, 0.23324957489967346, -0.5387265086174011, 1.5348294973373413, 1.7506531476974487, 1.1240907907485962, -2.049557685852051, 0.6434980630874634, -0.1560470163822174, -0.1457972228527069, -1.043588638305664, -0.6595973968505859, -2.314896583557129, 0.46733173727989197, -0.5582420229911804, 0.21458332240581512, -0.8498745560646057, -0.09388857334852219, -0.6963643431663513, 1.7051721811294556, -0.5245540738105774, 0.9314509630203247, -0.9747203588485718, -0.0321199968457222, -2.4748713970184326, -0.0783558040857315, 0.24973967671394348, -0.3325425088405609, -1.5126612186431885, 0.5861268043518066, -0.4907742142677307, 0.11182387918233871, -0.08055093139410019, -0.18604633212089539, 1.275226354598999, 0.3788183033466339, 2.4290432929992676, 1.687334418296814, -0.925891637802124, 0.19380441308021545, -1.1179715394973755, 0.7349164485931396, 1.4950772523880005, 0.04456824064254761, 0.7770729660987854, -1.0410189628601074, 1.1799423694610596, 1.4395748376846313, -0.1091122254729271, -2.9367737770080566, -0.6279196739196777, -0.11193443089723587, -0.37017518281936646], [1.7020764350891113, 0.20111185312271118, 1.1393376588821411, 0.5222718715667725, -0.6081077456474304, -0.38750338554382324, 0.12621887028217316, -0.507067859172821, 0.21241861581802368, 1.4614498615264893, 0.20493675768375397, 0.11265832185745239, -0.03277835249900818, -1.609499216079712, -0.0724412202835083, -0.7373928427696228, -0.3731304705142975, 1.1697924137115479, 1.0373682975769043, -1.0102242231369019, 0.5891607403755188, -0.7566428780555725, 0.01252166647464037, 0.29894694685935974, -1.1090062856674194, 0.1374373584985733, -0.4674084186553955, 0.8960124850273132, -0.70163494348526, -0.5156983137130737, -0.3302540183067322, 0.03979898616671562, -0.18897570669651031, 0.5553402900695801, -1.385634183883667, -1.0978028774261475, -1.6120272874832153, 0.9491412043571472, 0.2643742859363556, 0.06728647649288177, -1.9108964204788208, -0.9557126760482788, 0.9132925868034363, 0.5694906115531921, -0.4937264919281006, -0.41223666071891785, -0.9616428017616272, 0.4038177728652954, 0.10060293227434158, 0.6878407001495361, -0.8642418384552002, 0.14420628547668457, 0.13297420740127563, 0.67006915807724, 0.39488324522972107, 1.7744132280349731, -0.21533887088298798, 0.40433356165885925, 1.1833828687667847, 1.3126729726791382, 0.0032134708017110825, 1.4012796878814697, 1.2412315607070923, -1.014945387840271], [0.01789620704948902, 1.4950295686721802, 1.298458218574524, -0.032548464834690094, -1.2941356897354126, -0.15865063667297363, 1.0628135204315186, 0.4262271523475647, 1.1409633159637451, -0.3444057106971741, 1.6841078996658325, 0.06315161287784576, 1.1518110036849976, -0.5515915155410767, 1.806504249572754, -0.2843732535839081, -0.2018071711063385, 0.11729826778173447, -1.067155122756958, -0.9338252544403076, 0.03776851296424866, -0.8497658967971802, 0.8151779770851135, 0.22792568802833557, 0.4061807096004486, 0.07349788397550583, -0.52671217918396, 0.10951436311006546, -0.26437002420425415, 1.136926531791687, -2.0198817253112793, -0.18831047415733337, -0.7718431949615479, -1.5749508142471313, -0.729077935218811, -1.4011746644973755, 0.680729329586029, 2.1997010707855225, -1.7100111246109009, 0.8231344223022461, 0.46626317501068115, 0.49988454580307007, 1.2298414707183838, -1.30583655834198, -0.6864515542984009, -0.4094688296318054, 0.5248990058898926, -0.6089389324188232, -2.0582165718078613, 0.3629577159881592, -1.4127142429351807, 0.606453537940979, -0.021247241646051407, -0.37316522002220154, -2.090642213821411, 1.2586239576339722, 0.20136268436908722, -0.47186145186424255, 0.005137079395353794, 0.12082470953464508, 0.1259884387254715, 0.25065141916275024, -2.466522216796875, -1.8954356908798218], [0.2790452241897583, 1.8620809316635132, -0.7469748258590698, -1.1338152885437012, 0.7922651767730713, 1.0057636499404907, -0.22261197865009308, 0.8778788447380066, 0.28592970967292786, 2.0107765197753906, 0.6495341658592224, 0.5726636052131653, 0.5427181124687195, 0.21702270209789276, -1.0196068286895752, -0.3253980278968811, 1.0242375135421753, -1.1690759658813477, 0.850843608379364, -1.073190689086914, 0.9449642896652222, -1.4989804029464722, -0.6503666639328003, 0.11400970816612244, 2.227728843688965, -0.594456672668457, -1.6360218524932861, 0.01033727265894413, 0.6058784127235413, 1.2722523212432861, -1.613493800163269, -0.9304333329200745, 0.40522128343582153, 0.8837583065032959, -0.7689926028251648, -0.5107415914535522, 0.17511402070522308, -1.3607945442199707, 0.1560332179069519, 1.9090474843978882, 0.6028482913970947, -0.49993762373924255, -1.218526005744934, -0.8769170641899109, -0.3284544050693512, -1.9714856147766113, -0.8754786252975464, -0.5035077929496765, -0.38274991512298584, 0.4140172600746155, -0.7341558337211609, 0.06731204688549042, -0.2521916627883911, 0.9087827801704407, -0.4445865750312805, 0.2557053565979004, 0.45492056012153625, -1.3641244173049927, -1.2311230897903442, 0.6901111006736755, 0.2672036588191986, 0.5911229848861694, -0.40655866265296936, -0.10431595891714096], [0.05690367519855499, 0.7914795279502869, -0.0920553207397461, -0.1891993284225464, 0.009403800591826439, -0.4197736382484436, 0.01417990867048502, 0.17170970141887665, -1.1615159511566162, 2.1668200492858887, -1.1700000762939453, 0.565165102481842, 0.830948531627655, 0.4593811333179474, -0.737622082233429, -1.211709976196289, -0.6388779282569885, -0.6406620144844055, 1.0605190992355347, -2.187562942504883, -0.7550708651542664, -1.2795021533966064, 3.0961849689483643, -0.1909128874540329, 0.8129688501358032, 0.16288013756275177, 0.22426874935626984, 0.6064348220825195, -0.13188564777374268, 1.1317205429077148, -0.8762460947036743, -0.44246044754981995, -0.1788979172706604, 1.1654480695724487, 0.00038922668318264186, -0.576698362827301, 1.8991700410842896, -0.23867672681808472, -1.320678949356079, 0.0019178915536031127, 0.19257701933383942, -0.1991920918226242, 1.0228432416915894, -1.2128244638442993, 0.5377184748649597, 0.4961468577384949, -0.2752770185470581, 2.1556828022003174, -0.8667410612106323, 0.7360496520996094, 0.7642020583152771, -1.0096811056137085, 1.99622642993927, -0.1965167075395584, 1.450727939605713, 1.818118929862976, 0.8173791170120239, -0.46685653924942017, 0.6395096182823181, -1.3431023359298706, 0.5297124981880188, -0.019612343981862068, 0.35074788331985474, 0.4979080557823181], [0.45339280366897583, -0.24195212125778198, -0.5285868644714355, 1.390228509902954, 0.40709370374679565, -0.538955807685852, -0.7934951186180115, -0.41794317960739136, -1.7968471050262451, 1.2935899496078491, -1.262298345565796, 2.0454487800598145, -0.34338515996932983, -0.33185333013534546, -2.5947930812835693, -0.9942688345909119, 0.5232052803039551, -0.6283746361732483, 1.4294534921646118, 0.46491268277168274, -0.3204307556152344, -0.26553475856781006, 1.1776567697525024, 0.9054215550422668, 0.5219823122024536, 1.1422677040100098, -1.596397042274475, -0.3717956840991974, -0.9948901534080505, -0.35663098096847534, 0.9085448384284973, 1.0968377590179443, 1.0579310655593872, -1.979985237121582, -1.8265398740768433, 1.133249282836914, 0.0757688358426094, -1.4909223318099976, 0.914927065372467, 0.22765964269638062, -0.37873294949531555, 1.0288101434707642, -0.2771817147731781, -0.5407776832580566, 0.447086364030838, 0.3992687165737152, 0.06103399395942688, -0.820440948009491, -2.1810755729675293, -2.028501510620117, 0.752149760723114, 0.8494946360588074, -0.26588818430900574, 0.659115731716156, -1.1908597946166992, 1.3638290166854858, 0.12488309293985367, 1.1765321493148804, 1.5904910564422607, 1.1647322177886963, 0.7166004180908203, -0.5156911015510559, -0.837152898311615, 0.47530558705329895], [-1.108913779258728, 0.08972933888435364, -0.7176771759986877, 2.2690117359161377, -0.38061413168907166, -1.08394193649292, -1.3540940284729004, -0.8932517766952515, 0.8254656791687012, 0.7861961722373962, 0.900970995426178, 1.0125491619110107, -0.32690513134002686, -0.5960830450057983, -0.6247637867927551, 0.23669487237930298, 0.15603885054588318, -0.5629405975341797, 0.9759004712104797, 0.8471384048461914, -1.0493428707122803, -0.6571897864341736, -1.5112320184707642, -0.04518880322575569, -0.9086654782295227, 0.9716458916664124, -0.7325600981712341, 0.80040043592453, -0.1602385938167572, -0.20559187233448029, 0.3906782269477844, 1.646178126335144, -0.42623472213745117, -0.8704590797424316, -1.495111107826233, -0.08228600770235062, -0.4831438958644867, -1.6285696029663086, -0.3914032578468323, -0.5877083539962769, -0.5340840220451355, 1.9388306140899658, 0.6225307583808899, -0.8754388689994812, 0.41549092531204224, 1.753578543663025, 0.5758018493652344, -0.7541114091873169, 0.5877463221549988, -0.8950144052505493, -1.2577033042907715, -0.3105354309082031, 0.4753379821777344, -0.36352112889289856, 0.3974449634552002, 2.1656572818756104, 0.644888162612915, 0.6179059147834778, -0.09998253732919693, -0.7596203088760376, 0.46109041571617126, -0.4717682898044586, -0.650987446308136, 1.534173607826233], [-0.883225679397583, -1.6376776695251465, -0.27623531222343445, 1.017895221710205, 1.4227099418640137, -0.3073839843273163, -1.0802243947982788, -1.8730361461639404, 0.5824748277664185, 0.4068453013896942, 1.6425747871398926, 0.20641081035137177, -0.004573950543999672, 1.1000933647155762, -1.5405946969985962, 0.8006302714347839, -0.9446752667427063, 1.1760417222976685, 0.22183199226856232, 0.151176318526268, 0.21128331124782562, -1.257986068725586, 0.6263247728347778, -0.5422035455703735, 0.2382577657699585, -0.5412469506263733, 0.9667385220527649, 0.3199315071105957, 0.7056386470794678, -0.5481807589530945, -0.07007008045911789, -0.32901936769485474, -1.2337816953659058, -1.528651475906372, -0.3597731292247772, 0.5159862041473389, 0.41391721367836, -0.011186552233994007, 1.052912950515747, -0.8868441581726074, -0.466906875371933, -1.4169508218765259, 0.7859978079795837, -0.08792409300804138, 1.2794270515441895, -0.798376202583313, -1.2683789730072021, -0.2968502640724182, 1.7364171743392944, 0.715403139591217, -0.6985923647880554, -0.9687241315841675, -0.9174656271934509, 0.29285669326782227, -0.03180089220404625, -0.30802851915359497, -0.18988947570323944, -0.6087511777877808, 0.6218888759613037, 1.3609758615493774, -0.12730233371257782, 1.6946032047271729, 1.0564645528793335, -0.30387166142463684], [0.8263183832168579, 0.06464263796806335, -0.9176713228225708, 0.5735391974449158, 2.3224477767944336, 1.1473263502120972, -0.19611074030399323, 0.2687157690525055, -0.44388437271118164, 0.5547532439231873, 0.5902261137962341, -2.379157304763794, -2.096298933029175, -0.18273617327213287, -0.9338748455047607, -0.8037663102149963, -0.6089257001876831, 0.29043397307395935, 0.6729826331138611, 1.837728500366211, -1.2202175855636597, -1.7486097812652588, 0.7645239233970642, -0.9488044381141663, 1.2702924013137817, -0.40428999066352844, 0.2342166304588318, 0.2189694344997406, 0.5933409929275513, -2.0304999351501465, -0.17926116287708282, 0.6230360269546509, 0.8947315216064453, 0.6573440432548523, -1.3611650466918945, -0.6110672354698181, -0.386405885219574, 1.102945327758789, 2.0228676795959473, 1.057632327079773, -1.0879570245742798, -0.2805035412311554, -0.5205807685852051, -0.2130870819091797, 1.308782935142517, -0.38773009181022644, -1.3650227785110474, -0.5266615152359009, -0.37574175000190735, -0.7475510835647583, -0.7326042652130127, 1.7047830820083618, -0.32821032404899597, -0.8588159680366516, -1.2504453659057617, 0.8977661728858948, 2.507411241531372, 0.879039466381073, 1.61548912525177, -2.080312490463257, 0.42081359028816223, -0.42122969031333923, 0.12884379923343658, -1.2800548076629639], [0.23323781788349152, 0.07192730158567429, 0.006537769455462694, -0.88307124376297, 0.0019173725740984082, -0.537543773651123, 0.8225504755973816, 0.5414873361587524, 0.38628411293029785, -1.0906639099121094, -1.3371772766113281, -1.5141948461532593, -0.9155194163322449, -0.773146390914917, 0.9280697703361511, 0.8597157001495361, 0.2220865786075592, 0.11041388660669327, 0.46689078211784363, -0.7301484942436218, -1.2499674558639526, 2.0166327953338623, -0.5269697308540344, 0.19927547872066498, -0.9402806162834167, -0.34849604964256287, 0.5153030753135681, 0.7862198352813721, -0.03565843030810356, -0.7597765922546387, -2.532222270965576, -0.11513100564479828, 0.027927884832024574, -0.6389952898025513, 0.3886760473251343, -0.13944362103939056, 0.6029682755470276, -0.007590932305902243, 1.601693868637085, -0.0008153292001225054, 0.557137131690979, -1.1848119497299194, 0.4026106595993042, -0.6233097314834595, 0.8857015371322632, 1.5397547483444214, 1.5657840967178345, -0.9470531344413757, -1.4397414922714233, -0.2627449333667755, 0.22625796496868134, 1.0877331495285034, 0.16511127352714539, -0.39312443137168884, 0.5274899005889893, 1.9424715042114258, -0.2997056543827057, -1.0802950859069824, 0.9055070877075195, -0.7348127961158752, 0.23345747590065002, 0.4891985356807709, 0.033149246126413345, -0.08830618113279343], [-0.7678036689758301, 0.1846095770597458, 0.8667103052139282, 1.9267656803131104, -0.14597254991531372, 0.5964037775993347, 0.08492014557123184, -0.8823295831680298, -0.593332827091217, 0.3641047477722168, 0.2116512805223465, -0.6905642151832581, 0.15004441142082214, -0.33902451395988464, -2.2369143962860107, -0.3748447000980377, -0.3175623416900635, 2.0364835262298584, -0.8801792860031128, -1.5701148509979248, -0.16844117641448975, 0.016625775024294853, -0.27782461047172546, -1.6007256507873535, 0.5204352736473083, -0.3876079320907593, -1.1608805656433105, -0.5831559896469116, -0.3569524884223938, -0.17846150696277618, 0.9617093801498413, -0.008485918864607811, 0.7227367162704468, -1.7712377309799194, 2.827924966812134, -1.0937498807907104, -1.2521947622299194, -0.6525694727897644, 1.032607913017273, 0.7642611265182495, 0.5995281934738159, 0.39030641317367554, 1.231201410293579, -1.6529752016067505, -0.2272491157054901, 1.0578943490982056, 0.46008366346359253, -1.574476957321167, 1.082554817199707, 0.6362165212631226, -1.398066759109497, 0.24269652366638184, -0.5022833347320557, 0.5404905080795288, -1.5704371929168701, 2.303790330886841, 0.5920130610466003, 0.5549099445343018, -0.5949574708938599, -0.1981530636548996, 0.8726494908332825, -0.9740999937057495, -2.321108341217041, -1.1263575553894043], [0.6133254766464233, -0.3118613064289093, -0.9715129733085632, -1.2347501516342163, -0.8792787194252014, 0.693757951259613, 1.154615044593811, -0.33183321356773376, -1.4150105714797974, 0.11241161078214645, -0.0660325214266777, 0.4810057580471039, -0.011656242422759533, 2.1046533584594727, -0.328465074300766, 2.146646738052368, -0.029292751103639603, -0.29535195231437683, 0.6773350834846497, 0.4436590075492859, 0.625577986240387, -1.7164278030395508, -0.08724004030227661, 0.6026467084884644, 0.6560589671134949, 1.1389862298965454, -2.173856735229492, -0.21260786056518555, 0.09060412645339966, -0.729793131351471, -1.9543077945709229, -0.013561212457716465, 0.6919138431549072, 0.5672087073326111, 0.8914293050765991, 0.3171897232532501, -0.8377998471260071, 1.3118535280227661, -1.0460448265075684, 0.7928701043128967, 0.17861856520175934, 0.057222429662942886, 0.7531194090843201, -1.5894261598587036, -1.8280175924301147, 1.7137407064437866, -0.5439082980155945, -1.5745896100997925, -1.8911170959472656, -0.5437391996383667, -0.6110656261444092, 0.24338845908641815, -1.0248000621795654, -0.16188126802444458, -0.19975264370441437, -0.6961456537246704, 0.1108027994632721, -1.382804274559021, -0.4299551844596863, 2.7858963012695312, 1.9753917455673218, -1.5705476999282837, 0.4762316644191742, 1.1293202638626099], [0.8174982070922852, -0.32616090774536133, -1.7688469886779785, -1.1141915321350098, 1.1650192737579346, -0.520037829875946, -0.3438318073749542, 2.9023008346557617, -0.3018991947174072, 2.9254307746887207, 1.0965749025344849, -0.8419467210769653, -0.8052286505699158, 2.0556752681732178, 0.039857614785432816, 0.5442707538604736, 0.9570634961128235, 0.3148109018802643, -0.25474855303764343, 0.1644001603126526, 0.03171636909246445, 1.3804601430892944, -0.0461307093501091, -0.18340039253234863, -0.1286575198173523, -0.15302926301956177, -0.8347455859184265, 0.2967185974121094, 1.1641701459884644, 1.45986807346344, 1.0925889015197754, -1.1595538854599, 0.22393843531608582, 0.09974499046802521, 2.313753604888916, -1.0377846956253052, 0.8298234939575195, -0.11473166942596436, 0.5095678567886353, 0.08720576763153076, -0.5066211819648743, -0.5294783115386963, -0.938453197479248, -0.31691500544548035, -0.9642429351806641, 0.5224795937538147, 0.1789073646068573, -0.2318805754184723, -0.6996181607246399, -0.43638285994529724, 0.17716281116008759, 0.9091246724128723, -0.7543541193008423, 0.577618420124054, 0.44361329078674316, 0.8670790195465088, 0.049673423171043396, -0.20624881982803345, -2.140907049179077, -0.4691070020198822, -0.19012124836444855, -0.3260526955127716, -0.7271591424942017, 0.26878002285957336], [-0.06218001991510391, 0.7585927248001099, -0.33858439326286316, 0.1414491832256317, -0.18419893085956573, 1.0251233577728271, -0.7471721172332764, 0.9589579701423645, -1.3935415744781494, -0.7298799753189087, -2.4964358806610107, -0.9591605067253113, -0.2540860176086426, 0.5130584239959717, -0.7084808349609375, -0.6821776032447815, 1.0086612701416016, -0.06151703745126724, 0.06416651606559753, -0.021516771987080574, 0.5570030808448792, -0.44415566325187683, 0.8955720067024231, -0.8132234215736389, -3.769148349761963, 0.08603695034980774, 0.7977253794670105, 1.1582441329956055, -1.0506088733673096, 1.1521494388580322, -0.9531825184822083, 0.7629451155662537, -1.1568087339401245, 0.7273638248443604, -2.0453996658325195, -0.5349375009536743, -1.2840404510498047, -1.4158459901809692, -0.2074194997549057, 0.33524084091186523, -0.7834380865097046, -1.6514240503311157, -1.7561546564102173, 1.2009313106536865, 1.8367456197738647, 1.72597074508667, 0.21650642156600952, -0.1567794382572174, 0.5175970196723938, 1.3939944505691528, -1.5931649208068848, 0.5067115426063538, -0.5794609785079956, -1.2624609470367432, 0.0633506178855896, 0.08382884413003922, -0.3738017976284027, 1.540079116821289, 0.9815053939819336, 2.168389320373535, -1.0777239799499512, -0.767381489276886, 0.3959472179412842, -1.0053985118865967], [2.040370464324951, -1.128206729888916, 0.14363883435726166, 0.8006494641304016, -1.013957142829895, 0.6035046577453613, -0.11626552045345306, -0.08917627483606339, 0.39074885845184326, 0.8698720335960388, -1.1383665800094604, 0.9198020100593567, 0.055523306131362915, -0.6518848538398743, -0.05509716644883156, -0.7973939776420593, -0.30712890625, 0.6301471590995789, 0.9596920013427734, -0.6325342655181885, -0.394999623298645, -0.2600175738334656, -0.11825890094041824, 1.469317078590393, -0.17009170353412628, -1.2611538171768188, 2.028972864151001, 0.2979786694049835, -0.567369282245636, 0.319014310836792, 1.2664196491241455, 1.1202549934387207, -0.04593614488840103, 0.5636175274848938, -0.5238314270973206, -0.1010543704032898, 1.2938824892044067, 0.10731121897697449, -1.0504459142684937, 0.6093422174453735, -0.7589715719223022, 0.4501705765724182, -0.015201295726001263, -0.26354509592056274, -0.2709338665008545, -0.4607926607131958, 2.1137542724609375, -0.7390959858894348, -0.3446981608867645, 0.19879810512065887, -0.5521960854530334, 1.4355745315551758, 1.0831037759780884, 0.405439168214798, -0.8431993126869202, -0.3090420067310333, -0.2583686113357544, 0.4960867166519165, 0.618899405002594, -0.4460511803627014, 1.1412309408187866, -1.66211998462677, -0.4106297194957733, 0.649593710899353], [-0.2928670048713684, 0.10891936719417572, 0.5224581956863403, -0.5890788435935974, -1.794883131980896, 0.7362629175186157, -0.9445967674255371, -0.35080093145370483, -0.36265307664871216, -0.03914361074566841, -1.1589277982711792, -0.31245774030685425, 0.6291179656982422, 0.4099939167499542, -0.6090352535247803, 0.2547641098499298, -1.1854937076568604, -0.16536164283752441, -1.7069796323776245, -0.7196981906890869, -0.02675013616681099, -0.11788882315158844, -0.7961516380310059, 1.152943730354309, -1.5335631370544434, -1.3112905025482178, -1.5239524841308594, -0.5411781072616577, 0.5824244022369385, -0.03096587210893631, 1.807340383529663, 1.4348089694976807, 1.0600957870483398, 0.3173706829547882, -1.2261583805084229, -0.3158216178417206, 0.07513940334320068, -0.6198452115058899, -1.7641515731811523, 0.7548827528953552, -0.06661618500947952, -0.22893044352531433, 0.31237560510635376, 1.5780161619186401, 0.5574761033058167, -0.22423705458641052, -1.2566494941711426, 0.7391183972358704, 0.5694929361343384, -1.457031011581421, 0.561043918132782, 0.2691912353038788, 0.07517413794994354, 1.0472068786621094, -0.24532203376293182, -0.7265447974205017, -0.16722628474235535, -0.0647340640425682, 1.0186816453933716, -0.5495797991752625, 0.6173151135444641, -0.4650522768497467, 0.8725782036781311, 0.6104639768600464], [-0.3133178651332855, -0.6776105761528015, 0.356534868478775, 0.0169597826898098, -1.8649766445159912, 0.5669531226158142, 0.9970762729644775, 0.18382932245731354, 1.0526889562606812, 0.5153636336326599, 1.15109121799469, 0.6508029699325562, 0.6770450472831726, -1.0752884149551392, 0.10001711547374725, 2.456827402114868, -0.9976085424423218, -0.010475222021341324, 2.1313564777374268, -0.5869072079658508, 0.047010838985443115, -0.38525474071502686, 0.9525991082191467, -0.5287922024726868, 0.8530603051185608, -0.6111155152320862, -0.9931241869926453, 1.0978139638900757, -0.7037827372550964, 0.3668927550315857, 0.8033072352409363, 0.6725278496742249, -0.47149884700775146, -1.3628010749816895, 1.129695177078247, -0.2913641035556793, -0.07397692650556564, 0.16493238508701324, -0.2089982032775879, -1.1390618085861206, -0.12020743638277054, -0.5991770625114441, 0.6176028251647949, 1.3161617517471313, -0.5765560269355774, -0.6389161348342896, -1.2050824165344238, -0.5664695501327515, 0.9840139150619507, 0.3660105764865875, 0.05331050977110863, -1.1324756145477295, 0.21022440493106842, 0.4637473523616791, -0.4252602159976959, 0.79928058385849, -0.018086718395352364, -0.17868128418922424, -0.32817134261131287, -2.1746952533721924, -0.04104246944189072, 1.612412452697754, 0.4153522551059723, 1.2276448011398315], [-0.2663846015930176, -1.165317177772522, 1.4709049463272095, 0.8044466376304626, 1.1382198333740234, -0.21508190035820007, -0.23902568221092224, -0.21882861852645874, -1.4766851663589478, 0.29491710662841797, 1.567143201828003, 1.185391902923584, -0.20003457367420197, -0.17333580553531647, 1.861536979675293, -0.2031145542860031, -0.9944683909416199, 0.5531023144721985, -0.513241171836853, 1.0052831172943115, 0.0665484219789505, 0.7485841512680054, 0.3842022120952606, -0.8667247295379639, -0.27272066473960876, -0.29514017701148987, -1.9252049922943115, 1.365520715713501, 0.7619444727897644, -0.10848181694746017, -0.1742011159658432, 0.3234328329563141, -0.006785908713936806, -0.2101612538099289, -0.4387155771255493, 1.4908294677734375, -0.9606431126594543, -2.317349672317505, 0.8955048322677612, -1.354427456855774, -1.0946310758590698, 0.1941557675600052, -0.5285502672195435, 0.44081711769104004, -0.08827061206102371, 0.30698853731155396, -0.7046904563903809, -1.0439661741256714, 0.22373852133750916, -0.38000810146331787, 0.6967068314552307, -0.8567549586296082, -0.35902976989746094, -0.6220481991767883, -1.451974868774414, 0.22330592572689056, 0.1570775806903839, -0.6354273557662964, 0.2552083432674408, -1.6766375303268433, 0.9425561428070068, 0.732445240020752, -1.4484343528747559, -0.8115279078483582], [1.6611961126327515, -1.0981932878494263, 0.42428696155548096, -1.1081876754760742, 0.3195526599884033, -1.1338245868682861, 0.6201571226119995, -0.9193509221076965, -0.28425779938697815, -0.09549305588006973, -0.4234415292739868, -0.01548337284475565, -0.5121285319328308, 1.0373986959457397, -0.7793666124343872, -0.4445144236087799, 1.1389967203140259, -0.09793545305728912, -1.0205961465835571, 1.2688089609146118, -0.06939104199409485, -0.5935285687446594, 0.7384015917778015, 0.43322616815567017, -1.5569599866867065, 0.40428856015205383, 0.5500041246414185, 0.4742923974990845, -0.19765672087669373, -0.3145427703857422, 0.7494780421257019, 1.1788890361785889, -0.5445113778114319, -0.5400488972663879, -0.3150433301925659, 0.8101134896278381, 1.8830915689468384, -0.3718359172344208, 1.1271274089813232, -2.711601734161377, -0.30666643381118774, -2.739795446395874, 1.4169838428497314, 0.5416825413703918, 2.279674530029297, 0.501179039478302, -0.35338467359542847, 0.36330822110176086, 0.12763194739818573, 0.41364458203315735, 1.7843601703643799, 0.8848423957824707, 0.9179019331932068, 1.5744876861572266, -1.3110697269439697, -0.14671270549297333, -2.4631073474884033, 0.47506192326545715, -1.4194505214691162, -1.5565129518508911, -0.19722065329551697, -0.1897699236869812, -0.8474865555763245, -1.1091874837875366], [-1.2909246683120728, -0.05462726578116417, 0.9138861298561096, -1.0731905698776245, -1.8599270582199097, 1.8917981386184692, -0.3960988223552704, 0.1505742073059082, -0.6193978786468506, -0.039218030869960785, -0.07944542914628983, -0.02818235009908676, 0.7933972477912903, -0.9223348498344421, -0.06077440455555916, -1.4945118427276611, -1.5855109691619873, 1.9725228548049927, -1.5228660106658936, 0.9419689774513245, 1.2687032222747803, -1.6886197328567505, -0.5394688844680786, -0.9694200158119202, -1.0438281297683716, -0.378695547580719, 0.4938809275627136, 0.4343707859516144, -0.7255061268806458, 1.535204529762268, 0.6844797134399414, 1.2986969947814941, -1.5475422143936157, 1.0425190925598145, -0.873316764831543, -0.44498735666275024, -0.6179304718971252, 0.09146799892187119, -0.5376130938529968, 1.2461165189743042, 0.5159271359443665, 2.102386713027954, -0.6898260116577148, 2.074852705001831, 0.22565311193466187, -0.5342649817466736, -0.6682577133178711, 1.1058181524276733, 1.464004635810852, 1.1549770832061768, -0.005823296960443258, 0.12255322188138962, -0.6661877632141113, -0.7335836291313171, -0.24032986164093018, 1.0223405361175537, -1.066299557685852, -0.19376246631145477, 0.6792418956756592, -0.5770694613456726, -0.022329987958073616, 0.16655345261096954, -0.8726003766059875, 0.7023163437843323], [0.40381935238838196, -2.4046554565429688, 2.5251171588897705, 0.8450469970703125, -1.5338577032089233, -0.0032890615984797478, -0.23152810335159302, -0.15126949548721313, -1.5453696250915527, -2.007420301437378, -0.2269008308649063, -1.4146292209625244, 1.6295726299285889, 0.06475480645895004, -0.7307729721069336, -0.011031110771000385, 1.6889957189559937, -0.17892684042453766, 0.38010963797569275, 0.1174987331032753, -1.0361136198043823, 0.45574358105659485, 0.8942372798919678, 0.48084700107574463, -0.5484139323234558, 0.6444198489189148, 1.4613687992095947, -1.39980947971344, 2.116081953048706, -0.27720868587493896, -0.8380734324455261, -0.4664979577064514, -0.05709049105644226, 0.41619449853897095, 0.5400925874710083, 0.22765910625457764, -0.7594557404518127, 1.9876495599746704, 0.44772306084632874, 0.1554216593503952, 1.5189201831817627, 0.8434101939201355, 1.0086320638656616, 0.08655183762311935, -0.5599982738494873, 0.6943009495735168, -0.09241825342178345, 0.5562340617179871, 0.21651773154735565, -0.7314547300338745, 1.1079888343811035, 0.6089310646057129, 0.5516733527183533, 1.9646135568618774, 1.3110977411270142, 1.7208914756774902, 0.37393099069595337, -2.8725244998931885, -0.7527360320091248, -1.2879064083099365, 2.1112518310546875, 1.0277591943740845, 1.1553932428359985, 0.1983138769865036], [-0.3260931670665741, -0.6931740641593933, 0.017353925853967667, -0.6914182901382446, 0.3263002932071686, -0.22169624269008636, -0.887717068195343, 1.3485450744628906, -1.330214500427246, 0.06771843135356903, 0.8528504967689514, 0.6935769319534302, -0.07683948427438736, 0.5094922184944153, -1.0522067546844482, 0.34080275893211365, 1.1146519184112549, 1.2055944204330444, 0.7823435664176941, -1.6638702154159546, -1.2385767698287964, -0.5867284536361694, -0.2651672959327698, -0.1537424921989441, -0.4605206847190857, 0.779913067817688, -1.8992648124694824, 0.020939139649271965, -0.3477022349834442, -2.3493454456329346, -2.3505473136901855, 1.3280595541000366, 0.07631141692399979, -2.3647170066833496, -1.2810347080230713, -1.0773670673370361, 0.04884360730648041, 0.786395251750946, -2.7440690994262695, 2.5887222290039062, -0.3924795389175415, 0.46520835161209106, -0.8522056937217712, -0.5159087777137756, -0.25420013070106506, -1.4750269651412964, 0.5793206095695496, 0.6359100341796875, -1.7842583656311035, -0.15714795887470245, -0.974627673625946, 0.30856853723526, -1.201986312866211, -1.2820839881896973, -0.6365513205528259, 0.40433579683303833, 1.1488388776779175, 0.26015445590019226, -0.6846853494644165, 0.3260953426361084, 2.0848793983459473, 1.0076427459716797, -1.2188152074813843, 0.13159076869487762], [0.9393084645271301, 0.3407789468765259, 0.2022615522146225, 2.001934766769409, 1.4628576040267944, 0.43881943821907043, 1.0401581525802612, -0.030423514544963837, 0.5916914939880371, -0.639337420463562, -1.9978430271148682, -1.5977773666381836, -1.4619817733764648, -0.310398131608963, -2.733051300048828, -0.10392767935991287, -1.7044566869735718, 1.711926817893982, -0.054665468633174896, -1.0760360956192017, -0.7851057052612305, -0.6200066208839417, 0.857582151889801, 0.1802178919315338, -0.27272501587867737, -1.4943556785583496, 0.12135165184736252, 1.4750217199325562, -0.14886076748371124, 0.4310881495475769, 0.09597894549369812, -0.3350519835948944, 0.767899215221405, -0.2786668539047241, -1.7197514772415161, 0.2816845774650574, 0.8617615103721619, 0.14042925834655762, 2.434847831726074, -1.4903281927108765, 0.1851436346769333, 0.9880306720733643, -0.23521308600902557, 1.1400598287582397, 1.1574877500534058, 1.590657353401184, 0.7023686766624451, 0.11215520650148392, -1.188288688659668, -1.4893131256103516, 0.8133365511894226, 0.35442033410072327, -0.8323390483856201, -0.2753114700317383, -0.7571491003036499, -0.44430238008499146, 0.6952559351921082, -0.0044058384373784065, 0.08920372277498245, -0.8716843128204346, -0.4061265289783478, 0.6988381743431091, 0.5483016967773438, -0.954864501953125], [1.1481297016143799, 0.9798718690872192, -0.6192899942398071, 1.1296812295913696, 1.0301258563995361, -0.47781768441200256, -0.9509125351905823, -0.32302501797676086, 0.16877514123916626, -0.8182771801948547, 0.48602184653282166, -0.6709010601043701, -0.8601142764091492, 0.5442304015159607, 0.11761109530925751, -0.9241413474082947, 1.3419911861419678, -0.38308045268058777, 0.07830559462308884, 2.2191169261932373, 1.1570558547973633, 2.6199326515197754, -1.6641579866409302, -0.5442887544631958, 2.6381654739379883, -1.1113966703414917, 1.4291222095489502, -1.190955400466919, -0.3068103492259979, -0.1756235659122467, 1.571515679359436, -0.40803471207618713, -1.8282772302627563, 0.5605933666229248, 0.11950155347585678, 0.10650335997343063, -0.3969320058822632, 2.1007444858551025, -0.1177562028169632, 1.5943787097930908, -1.2584178447723389, -0.024659913033246994, 0.39879173040390015, 0.12232671678066254, -1.377160668373108, 0.6164165735244751, 2.131288766860962, -1.0887494087219238, -0.5595976710319519, -0.2081356793642044, -0.7208467125892639, -0.9844849109649658, 0.32871946692466736, -0.8114562034606934, 0.6560003161430359, 0.9638007283210754, 0.36524859070777893, 0.3591587245464325, 1.3357868194580078, 1.8735145330429077, -0.7571697235107422, -0.2565491199493408, 1.773593544960022, 0.14528976380825043], [0.42859330773353577, 0.6068278551101685, 2.2498719692230225, -0.6799027919769287, -0.47831305861473083, -0.812348484992981, -0.33039218187332153, -0.7013610601425171, 3.8891379833221436, -0.077255979180336, 0.5477426052093506, -0.6949324011802673, 0.3564966320991516, -1.9549974203109741, 0.9612599611282349, -0.238722026348114, 0.6680295467376709, -0.03326128423213959, -0.6372409462928772, -0.8438524007797241, -0.20903600752353668, 0.051930345594882965, 0.17677289247512817, 1.1896191835403442, 1.269782304763794, -0.49040743708610535, -1.6183222532272339, -0.9902003407478333, 1.295072078704834, 1.198826789855957, -0.3265116512775421, 0.6078280210494995, 0.30659812688827515, 1.180753469467163, 0.6615541577339172, 0.5492538809776306, -0.43250688910484314, -0.5446687936782837, -0.07460886985063553, 0.599149227142334, 0.7490106821060181, 2.0530760288238525, -0.29321005940437317, 1.137206792831421, 0.2548607289791107, 0.5976997017860413, 0.265018105506897, -0.5123420357704163, -0.3537907600402832, 1.2043522596359253, -0.7235206365585327, 0.3950267732143402, 1.17649507522583, -0.17312966287136078, -0.6900159120559692, 0.7419759631156921, -0.5453373789787292, 0.4579102396965027, 1.7244845628738403, 1.8352789878845215, -1.0320096015930176, -0.6098647713661194, -0.3568960726261139, 0.39247408509254456], [0.3678975999355316, -0.5763018131256104, 0.25754231214523315, -0.0380619578063488, 1.7008663415908813, 0.918749213218689, -0.8052471876144409, -1.1624367237091064, 0.10934136062860489, 1.7816331386566162, -0.7628541588783264, 0.07173646986484528, -0.6638163328170776, -0.23068492114543915, -1.1138379573822021, 0.9336189031600952, -2.313478946685791, -1.6242095232009888, -0.601579487323761, -0.9926574230194092, 0.35876330733299255, -0.3928690254688263, -1.2754319906234741, -0.18179760873317719, 2.1155765056610107, -0.1337282657623291, -0.8118004202842712, -1.310957431793213, 0.7078937292098999, -0.8193051815032959, 0.1929323822259903, 0.43823179602622986, 0.6084033846855164, -1.0656646490097046, 0.7156628966331482, -1.827894926071167, -0.9547531604766846, 0.30707359313964844, 0.28056764602661133, -0.799791693687439, 0.9550822973251343, -1.3039878606796265, 0.12004489451646805, -0.9253025650978088, 0.21031156182289124, 0.15595261752605438, -0.49751704931259155, 1.1399277448654175, -0.4333595931529999, 0.0916212722659111, 0.0210221316665411, -0.9099895358085632, 0.3340310752391815, -2.088000774383545, 0.3291236460208893, 1.3109095096588135, 0.5730748772621155, -0.8465850353240967, 1.6504892110824585, 0.247574120759964, 1.631681203842163, 0.7812767624855042, 0.8398672342300415, -0.747155487537384], [-0.5796326398849487, -0.8048720955848694, 0.2848968505859375, -0.03646061569452286, 1.3887715339660645, -1.1249964237213135, -0.3455832898616791, 1.161017656326294, 0.5831863284111023, 0.10145207494497299, -0.09917005896568298, -0.29658523201942444, -2.209129571914673, 1.1033260822296143, -1.8761711120605469, 0.5827754735946655, -0.8659931421279907, 1.029018521308899, 0.1297936737537384, 1.2150146961212158, 2.112567186355591, 0.4860609769821167, -0.15590186417102814, 1.2329832315444946, 0.33088594675064087, 0.20541124045848846, 0.8010583519935608, 0.44542476534843445, -0.9684536457061768, 0.4897178113460541, -1.6312259435653687, -2.2783398628234863, -1.3436301946640015, -0.8689601421356201, 0.6912322044372559, 1.217828392982483, -1.1474425792694092, -0.28961899876594543, 0.03112971968948841, -0.12079032510519028, -0.2537623345851898, 0.1936928927898407, -0.5999897718429565, -0.8626559376716614, 0.2410942167043686, 0.0808618813753128, -0.44148826599121094, 2.602888345718384, -1.281576156616211, -0.3998155891895294, 0.92170250415802, 0.02712377719581127, -0.2023942768573761, 0.8190248012542725, 0.09658932685852051, 0.6345932483673096, 1.0883362293243408, 0.4633735716342926, 0.18764561414718628, -2.6824839115142822, 0.4766664206981659, 0.7008860111236572, 0.10142811387777328, 0.6193925142288208], [0.6511116623878479, -1.30706787109375, 0.4686916768550873, 1.7420670986175537, 0.5610109567642212, 0.5695160627365112, 1.7021673917770386, 1.5804023742675781, 0.6006232500076294, -0.1428992599248886, -0.8789802193641663, -0.662839412689209, -0.04452676698565483, 0.874279797077179, 1.88511323928833, -1.5136317014694214, 0.0067143221385777, 0.5217398405075073, 1.5052858591079712, -0.394339919090271, -0.6093534827232361, -0.2021532952785492, 1.2461367845535278, 0.07620763778686523, 1.1292847394943237, 0.014037641696631908, -0.6612173318862915, -1.4781960248947144, 0.37079834938049316, -0.17763042449951172, 0.7618972659111023, -0.8672869205474854, 0.9828360080718994, 1.8108595609664917, 0.8622162938117981, 0.38489052653312683, 0.8966063857078552, -0.5395310521125793, -2.2637667655944824, 0.36796292662620544, 0.6809309720993042, 0.8733647465705872, -0.6479933857917786, -1.307450771331787, 0.5014715790748596, -0.8566091060638428, -0.1389816403388977, -1.2682907581329346, -1.121256709098816, -0.49874454736709595, -0.9873194694519043, -0.13378740847110748, 2.1562113761901855, -2.097517728805542, -2.247389554977417, 0.8142460584640503, 0.5761619210243225, -0.4131724238395691, 1.352060317993164, 0.8700831532478333, -0.36017975211143494, -1.9951889514923096, 0.8711674809455872, -0.5450434684753418], [1.398695707321167, 0.46148431301116943, 0.8673599362373352, 0.0036582876928150654, -2.0633625984191895, 1.0936851501464844, -2.724428653717041, 0.9979864358901978, -1.0621674060821533, -2.4168615341186523, 0.38141486048698425, 0.6527882814407349, -0.05748721957206726, -1.3512729406356812, 1.4464374780654907, 1.0668162107467651, -0.6656081080436707, -0.42231035232543945, -2.166759490966797, 0.6199737787246704, -0.643523633480072, -1.4599961042404175, 0.9711490869522095, -0.4136950373649597, 1.050779104232788, 1.0990707874298096, -1.3107595443725586, -0.8950912952423096, 0.8481273055076599, -0.9164186716079712, 1.8865501880645752, -1.2475011348724365, 0.0011978673283010721, -0.23122669756412506, 0.6654428839683533, 0.8591744303703308, -0.7965437173843384, -1.3894331455230713, 2.187243938446045, 1.7507728338241577, 0.0424656942486763, -0.5349182486534119, 2.3069067001342773, -0.4515162706375122, -1.535040259361267, 1.528317928314209, 1.0208406448364258, 0.2919439375400543, -0.5835195779800415, 2.46081280708313, -0.8785043358802795, 1.1375552415847778, 0.5776615738868713, 1.0498127937316895, -1.1158239841461182, 0.32815054059028625, 1.2333927154541016, 0.40760114789009094, -0.16255499422550201, -0.15151798725128174, -0.5448840260505676, -0.02698780596256256, -0.6180872321128845, -0.6838595271110535], [2.059542179107666, -1.8136268854141235, 0.6744121313095093, 0.0946316048502922, 0.09882227331399918, -1.00193190574646, 0.8045036196708679, -0.08547405898571014, -0.09968864917755127, 0.9755792617797852, 0.015942225232720375, -0.6260850429534912, -0.9440344572067261, 0.7047659158706665, 0.15902774035930634, -0.13006441295146942, 1.1312038898468018, 0.6706684231758118, 0.5963314175605774, 0.20225182175636292, 0.469752699136734, 1.2032479047775269, -1.2378344535827637, 1.531668782234192, 0.6430700421333313, 0.6456536650657654, -0.6313560605049133, 1.6527537107467651, 1.0754483938217163, -0.8513507843017578, -1.0990958213806152, -1.571963906288147, 0.6560389995574951, 0.15334288775920868, -1.0246567726135254, 0.05147101730108261, 0.025905825197696686, -0.6694945096969604, -0.7063034772872925, 0.41013890504837036, -0.9388677477836609, -2.0305583477020264, 1.9816237688064575, 0.5691819787025452, -1.241508960723877, -1.7116219997406006, -0.7195380330085754, 1.4069771766662598, 2.7707417011260986, 1.5345567464828491, 0.8651313185691833, 0.6238429546356201, 0.6955992579460144, -1.9325103759765625, 0.30861738324165344, -0.08031698316335678, 0.3965892791748047, 2.8004822731018066, -0.5979011058807373, 1.5402131080627441, 1.1055033206939697, 0.2133806049823761, -1.4833258390426636, -1.1472394466400146], [0.47752541303634644, -1.0991657972335815, 0.20942378044128418, -0.7040731906890869, -0.33857783675193787, 1.6649014949798584, -2.2995290756225586, -1.1310033798217773, -1.7828015089035034, 0.24349743127822876, -0.10927475243806839, -0.427663117647171, -0.4200739860534668, -1.6759964227676392, 0.7824452519416809, 0.19960352778434753, 1.4677000045776367, 0.1346067637205124, 0.4228014647960663, -0.96903395652771, 0.0898280143737793, 0.6273913383483887, 0.2060662806034088, -1.1371402740478516, -0.25891467928886414, 1.023047685623169, -1.7916172742843628, -0.6667864918708801, 0.6643476486206055, 0.09545190632343292, 0.32931193709373474, -1.1340925693511963, 0.053640954196453094, 0.14597605168819427, 0.4469870626926422, -0.15375752747058868, 1.8825569152832031, -1.5715479850769043, -0.47223201394081116, -0.6187461614608765, 0.8763781189918518, 0.8855463266372681, 1.1168889999389648, -1.649385690689087, 0.1667204052209854, 0.25152599811553955, -0.6548492908477783, -0.1914714276790619, -0.40092238783836365, 0.6863117814064026, -2.2037203311920166, -0.1804015040397644, -1.5307140350341797, -1.6621074676513672, -1.190797209739685, -1.12062668800354, -0.4683392345905304, 0.1347586214542389, 0.03581499308347702, 0.09271246194839478, -0.2878882884979248, 2.0587737560272217, -0.7587159872055054, -0.7715136408805847], [0.791970431804657, 1.0855752229690552, 1.1083881855010986, -0.3220204710960388, -0.7225281596183777, -1.3701947927474976, -0.7316072583198547, -0.42374783754348755, 0.7993760108947754, -1.0294269323349, 0.5299025177955627, -0.13062864542007446, 0.46738675236701965, 0.3378824293613434, -0.8044151663780212, -0.045132700353860855, -1.1904057264328003, 1.0383830070495605, 0.12614868581295013, -0.34124457836151123, -0.7996776700019836, -0.18694674968719482, 1.837437629699707, 0.020703734830021858, 2.0840160846710205, -0.6625919342041016, -0.7364640235900879, 1.3620715141296387, -0.4411814212799072, 1.0509426593780518, -0.9133602976799011, -1.0010408163070679, 1.206752061843872, -1.6900050640106201, -0.34799924492836, -0.08415817469358444, -0.006092578172683716, 1.4248539209365845, 0.6286532878875732, 1.2210255861282349, -0.7622475028038025, -0.3109003007411957, -0.7761910557746887, 0.24319706857204437, -1.0251743793487549, -0.6380512714385986, -1.0216337442398071, 0.4750857651233673, 0.7710815072059631, 1.117435336112976, 0.5878426432609558, -0.31466424465179443, 0.5946364402770996, -0.5495247840881348, 0.14993833005428314, -0.23823130130767822, 0.22363755106925964, 1.9193990230560303, 0.972895622253418, -0.7670964598655701, 1.053362488746643, 0.9486891031265259, 1.4945435523986816, 0.07520691305398941], [-1.1790107488632202, 0.38837894797325134, -0.05314703658223152, -1.0977674722671509, 0.14629852771759033, -0.5198960304260254, 0.6021578311920166, -0.45549046993255615, -0.441135972738266, -0.49476227164268494, -0.8388869166374207, 0.769623875617981, -0.2727016508579254, 0.04252435266971588, -0.4271065592765808, 1.5111923217773438, 0.48715996742248535, -0.5055793523788452, -1.2481054067611694, 0.6629807949066162, 0.9937307238578796, 1.414780616760254, -2.2330830097198486, -1.9256383180618286, 0.7010968923568726, 0.0016156523488461971, -1.6665029525756836, 2.4814083576202393, 2.2752416133880615, -0.7343616485595703, 1.002881407737732, 0.04614584520459175, -0.8536256551742554, 1.031413197517395, 0.8909193277359009, 0.13899651169776917, -0.5123183727264404, 1.0790603160858154, -1.33847975730896, -0.403237909078598, 0.7503138780593872, -1.912377953529358, 0.41086581349372864, -1.1530230045318604, -1.3965693712234497, -0.9983375072479248, 0.2287648469209671, 0.3405646085739136, -0.4609195291996002, -1.0413029193878174, 1.3508858680725098, 1.813390851020813, 1.013217806816101, -0.21725308895111084, 0.5241938829421997, -0.9918854236602783, 2.794811248779297, 0.32751116156578064, -1.0636193752288818, -0.2952287793159485, -0.8054302930831909, 0.6727052927017212, -0.6239464282989502, -0.19545526802539825], [0.1101541742682457, 1.1796094179153442, -0.6838213205337524, -0.17175355553627014, 1.627748727798462, 1.198983907699585, 0.061837512999773026, -0.767867922782898, 0.3695352375507355, 0.1360786259174347, -0.2951570749282837, 0.686501145362854, -0.5966607332229614, -1.544372797012329, 0.9036355018615723, 0.11563847959041595, 0.9678382277488708, -1.3231532573699951, -0.850212574005127, 0.48038533329963684, -0.788699209690094, 0.2044866383075714, 0.5005291700363159, 0.17776387929916382, -0.582295298576355, -0.7422367334365845, 0.44963976740837097, -0.7022908329963684, 1.0282478332519531, -1.3666058778762817, -0.005426282063126564, -0.8459703326225281, 1.0420957803726196, 0.8730319142341614, 1.2076914310455322, 0.3441290557384491, -2.854663848876953, -0.1600487232208252, 0.38961678743362427, 0.33766788244247437, 0.5477612018585205, 0.055199556052684784, 0.07389318943023682, 0.7555682063102722, 2.4919018745422363, -2.2590949535369873, 1.8796297311782837, -0.13438746333122253, -0.1208224818110466, -0.6526229977607727, -1.0408685207366943, -0.830207109451294, 1.0843870639801025, 0.19278299808502197, 1.4012876749038696, -2.059072256088257, 0.33148741722106934, -1.1063268184661865, -0.26603448390960693, -0.8041426539421082, -0.34810835123062134, -0.17971165478229523, 1.088820219039917, 0.18389558792114258], [-0.1496364325284958, 0.27568739652633667, 0.8442031741142273, -0.49619725346565247, -1.0570751428604126, 1.372680902481079, 0.5340293645858765, 0.7403413653373718, -0.02596951276063919, 1.0561319589614868, -0.2635742127895355, -1.4132903814315796, -0.3712063431739807, -0.0190983135253191, -1.2864644527435303, 0.4305790662765503, 0.22178353369235992, -0.6586562991142273, 0.13464081287384033, -0.2254878431558609, -0.5542158484458923, 1.093994379043579, -0.7279874682426453, -0.7165408730506897, 0.7251472473144531, 0.521647036075592, 0.8776508569717407, -1.3763782978057861, -1.6707701683044434, 0.47109493613243103, -0.2111862748861313, -1.0220844745635986, -0.24136964976787567, -0.9266330599784851, -1.0775740146636963, -0.02351153828203678, -0.19614724814891815, 0.5502560138702393, 1.2004272937774658, -0.6313189268112183, -1.2680935859680176, 0.30451640486717224, 0.4012680947780609, -0.9138349890708923, 0.5592321753501892, -0.06789842993021011, 0.0070904274471104145, 1.3528504371643066, 0.15422987937927246, -1.674165964126587, 0.7634027600288391, 0.33305075764656067, -0.4489854574203491, 1.3789886236190796, 1.3639030456542969, 1.2635148763656616, -0.3669739067554474, 0.5461123585700989, 0.6974441409111023, -1.248623013496399, -0.7604804635047913, -0.564765453338623, -0.3345358371734619, -0.44426092505455017], [-0.7339052557945251, -0.3672330677509308, -1.1985772848129272, 2.6355626583099365, -1.4722182750701904, 0.7632902264595032, -0.7390640377998352, -0.3545895218849182, -0.4723004996776581, 1.1596970558166504, -1.999186635017395, 0.7443314790725708, 0.4629242718219757, 0.647792637348175, 2.314237117767334, 0.39134836196899414, -0.01980104111135006, -1.7859059572219849, -0.15190176665782928, 0.7992327213287354, 0.5933228135108948, -0.9881576895713806, -1.5971688032150269, 0.8783847093582153, -0.245750293135643, 0.5705335736274719, 0.8064610958099365, 0.14270161092281342, 1.1746875047683716, 0.060138583183288574, -0.04164888337254524, 0.3282008171081543, -0.3674822747707367, 0.9821448922157288, 0.5796668529510498, -0.8248454928398132, -0.5263649225234985, 0.8721974492073059, -0.19800624251365662, 0.3852289319038391, 0.5731119513511658, -0.12120083719491959, -1.430687427520752, -0.48913073539733887, -1.2820963859558105, 0.17080844938755035, 0.5410427451133728, 2.54046893119812, 0.9155532717704773, -1.0865838527679443, -0.4677869379520416, -0.6407029628753662, 0.2564902901649475, 0.8954421281814575, 0.8711786270141602, 1.8414900302886963, -0.02700069360435009, -0.9770531058311462, -1.8942420482635498, -0.40364620089530945, -2.012385845184326, -0.13707995414733887, 0.4655713438987732, -0.6180886626243591], [-0.3987901508808136, -0.7120717167854309, -1.592300295829773, 1.0750670433044434, -0.3331851363182068, 0.321970671415329, -0.8052346706390381, 0.10565979033708572, 0.46758875250816345, 0.3932020962238312, -0.8020198345184326, -0.6038709878921509, 0.43816444277763367, -0.10741584002971649, 0.5151717662811279, 0.005943071097135544, -0.15498098731040955, -1.337766408920288, 0.13773371279239655, 0.8718459010124207, -0.54947429895401, -0.28436189889907837, -0.7025659680366516, 0.9105467200279236, -1.0045589208602905, -0.2845178246498108, 0.4763076901435852, -0.5214029550552368, -0.2920513153076172, -0.24867480993270874, 0.40208008885383606, 0.5820979475975037, -0.8256908059120178, 0.43720099329948425, 0.12869231402873993, 0.8364096879959106, 1.039391279220581, -0.0005697169690392911, -0.07204743474721909, -1.1909613609313965, 0.974860429763794, 0.40722233057022095, -0.8839735984802246, -0.6708478331565857, -1.668073296546936, 1.127326488494873, 0.5720078349113464, -0.19335409998893738, -2.217341661453247, 0.739743709564209, -2.7901291847229004, -0.08319389820098877, 0.8185886144638062, 1.36551833152771, -0.6598122715950012, -0.22658495604991913, 0.6280312538146973, -2.3705742359161377, 0.9307432174682617, 0.05298648029565811, -0.47429919242858887, 1.0843149423599243, 2.024880886077881, 0.3983192443847656], [0.9115400314331055, -0.6012911796569824, -1.308058500289917, -0.024020621553063393, 0.12177184224128723, -0.16132016479969025, 0.5356472730636597, -0.7273973822593689, -1.566377878189087, -1.7304106950759888, -0.800755500793457, -1.3363008499145508, -0.23703446984291077, 1.8428140878677368, 0.30566516518592834, 1.2517290115356445, 1.2999635934829712, 1.9182043075561523, 0.2733851373195648, -1.1498292684555054, -0.24702690541744232, -1.3657585382461548, 2.026146173477173, 0.254818856716156, -1.0909472703933716, -0.33315879106521606, 0.0420292392373085, -0.5333801507949829, -0.33849915862083435, -1.3146384954452515, -0.7070134878158569, 0.7605912685394287, -0.4416685998439789, -0.7243619561195374, -0.19015447795391083, 0.22160935401916504, 0.8658503890037537, -0.6368967890739441, 0.410036563873291, -0.7951487302780151, 0.053584132343530655, -1.189139723777771, 0.40321260690689087, -0.6544017791748047, -2.4992778301239014, 0.15068793296813965, -1.4225419759750366, -0.1616247445344925, -0.3804740905761719, -0.2938954532146454, 1.07016921043396, -0.08667164295911789, -0.7665894627571106, 0.7020094394683838, -0.7531463503837585, 0.7047160267829895, 0.2162618339061737, 0.7531126737594604, -0.2673886716365814, 0.9177689552307129, -0.5120124816894531, 0.07577218115329742, -0.5498369336128235, 1.6073848009109497], [1.459247350692749, -0.4610886871814728, 1.4451924562454224, 0.8090338706970215, -0.9946588277816772, 0.2020435780286789, 0.45158660411834717, -1.3601421117782593, -0.7841973900794983, -0.5121027827262878, -0.22384317219257355, -0.6929153203964233, -0.20226319134235382, -0.9315341114997864, 0.8252267837524414, -0.05742732807993889, 0.3018602132797241, 1.6720716953277588, -0.5569708347320557, -0.1829541176557541, 0.3248327076435089, -1.2323542833328247, -0.5697593688964844, 0.30570095777511597, -0.20674632489681244, -2.9054863452911377, -0.802945613861084, 1.3926780223846436, 0.17959731817245483, 0.2803923487663269, -0.6858307719230652, -0.48949047923088074, 0.22767707705497742, 1.164719581604004, -0.18806496262550354, -1.36738920211792, -0.5737788081169128, -2.6531574726104736, 1.077311635017395, -0.2617221176624298, 0.8617343306541443, 1.026509404182434, 0.02927115373313427, -0.7375163435935974, -1.0866585969924927, 0.2444542944431305, 1.0291080474853516, -0.49199050664901733, -1.053625464439392, 0.4702368676662445, 0.8206045031547546, 1.2352261543273926, -0.6995705366134644, -0.2569165527820587, 0.3255859613418579, 1.248494029045105, 1.6149883270263672, -0.28539109230041504, 0.1421036720275879, -1.268248438835144, -1.9623812437057495, -0.12177220731973648, -0.1999528855085373, -0.7124049663543701], [-2.027052164077759, -1.3492333889007568, 0.5624107718467712, 0.04929574951529503, 0.4744005799293518, 0.06090587377548218, -2.124997138977051, -0.5500145554542542, 0.3905300498008728, 0.11244217306375504, -2.031865119934082, -0.6730713844299316, -1.2927627563476562, -0.3051789700984955, 1.436546802520752, -1.4228928089141846, -0.006524009630084038, 0.24342292547225952, 0.6634283661842346, 1.1794531345367432, -0.3102010488510132, -0.6293091773986816, -0.7024713158607483, -0.24013836681842804, -0.6402671933174133, -0.47960707545280457, -0.0953502357006073, -0.4673275947570801, -1.3004359006881714, -0.3271371126174927, -0.792276918888092, 0.30334049463272095, 0.60256028175354, -0.6670520305633545, -0.10596220195293427, -0.3057432174682617, -0.5524060130119324, -0.40599963068962097, 1.748854398727417, 0.5116748213768005, -0.6363455653190613, -0.5518854856491089, 0.15195664763450623, 0.4031428098678589, -0.49584200978279114, -0.5795916318893433, 0.9734021425247192, -1.9631036520004272, 0.7126134037971497, 0.6531457901000977, 0.8778486847877502, -0.1830892115831375, -0.35625705122947693, -1.038421630859375, 1.406407356262207, -0.1820441335439682, 1.448839545249939, 0.7296915650367737, 1.1897696256637573, 0.09225841611623764, -0.21524502336978912, 0.238325297832489, -0.1673124134540558, 0.344779908657074], [0.6405996084213257, -0.1039513573050499, 0.2261570394039154, -0.22434523701667786, 0.41105934977531433, 1.0401535034179688, -1.1993892192840576, 1.5031192302703857, -0.6283331513404846, -1.6248310804367065, 1.4581104516983032, 0.49206772446632385, -0.11229106038808823, 1.061928391456604, 0.47973495721817017, -1.3524776697158813, 0.24141621589660645, 0.3685784339904785, -1.6563622951507568, -0.480330228805542, -0.055952273309230804, -0.4746582806110382, -0.41835999488830566, 0.9546527862548828, 0.6703991889953613, -0.059079475700855255, -0.9162837862968445, -0.02779468707740307, 0.9751407504081726, -0.9012677073478699, -0.5248035788536072, -0.8022116422653198, 0.4044486880302429, -0.4718082547187805, -0.32953494787216187, 0.38263362646102905, -0.6335684061050415, 0.28221797943115234, 0.031694915145635605, 0.2630373537540436, -0.1402972936630249, 0.7592757940292358, -0.7989177107810974, -0.2756690979003906, -1.3498173952102661, 0.28923654556274414, -0.5058306455612183, 0.8259330987930298, 0.8896496891975403, -0.11479675024747849, -0.19617201387882233, 0.16131877899169922, 0.5427254438400269, 0.3965360224246979, 0.15854144096374512, 1.9076532125473022, 0.6455006003379822, 0.037995051592588425, 0.5759568214416504, 1.4351803064346313, 0.36868730187416077, 1.5730847120285034, 0.913807213306427, 0.39197495579719543], [0.716852605342865, 0.049487680196762085, 0.049969907850027084, -0.1365688592195511, 0.6397420763969421, 1.0688483715057373, 0.2792050838470459, 1.2755792140960693, 0.028269458562135696, 1.4462757110595703, -0.7513414621353149, 1.3339202404022217, -1.229537010192871, -1.2033436298370361, 1.8110883235931396, 1.5534037351608276, 2.431614398956299, 0.7051674127578735, -0.036294013261795044, 2.3829143047332764, 0.9999034404754639, 0.2396887242794037, -0.07757745683193207, -0.8106399774551392, 0.7465565800666809, 0.7659049034118652, 1.9445339441299438, 0.3789420425891876, -0.36566582322120667, 1.1514348983764648, 0.08708468824625015, -2.3802645206451416, 0.5748118758201599, -0.7756113409996033, -1.5784786939620972, 0.6104536056518555, -2.693416118621826, -0.03356054052710533, -0.5179102420806885, -1.1711857318878174, 0.5363780856132507, -1.1060599088668823, -0.3315383195877075, 0.973495602607727, 0.9328449964523315, -1.2850918769836426, -0.24404828250408173, 0.39718154072761536, 2.0957083702087402, 1.397240400314331, 1.2377865314483643, 0.2324821799993515, -1.2938554286956787, -1.0487651824951172, -0.3490137457847595, -1.0305427312850952, 0.3567255139350891, -0.11244060844182968, 0.5033266544342041, -0.7215938568115234, 0.10818781703710556, 0.4921167492866516, 0.02055053971707821, 1.5131011009216309], [-0.5283249020576477, 0.2975202798843384, -1.3754314184188843, -0.4550018310546875, 0.5597396492958069, -0.05017371475696564, -1.3261727094650269, -0.05815942957997322, 0.3222775161266327, 0.17659199237823486, -0.46761080622673035, 1.0988755226135254, -0.4713423550128937, -0.27785998582839966, -1.3937989473342896, 0.6043027639389038, -3.209378719329834, -0.5934001207351685, 0.5929883718490601, -0.7315131425857544, -0.029487453401088715, -1.9864472150802612, -0.23658387362957, 0.20101185142993927, -1.0954054594039917, 0.5128617882728577, 0.3606146275997162, 0.7514625787734985, 1.0321788787841797, 0.229304701089859, -0.7767358422279358, -0.9209269285202026, 0.20623044669628143, -0.11779119074344635, 0.7216094136238098, -0.9186904430389404, 0.33235964179039, -1.0696277618408203, 0.2762737572193146, -0.06981519609689713, -1.3014447689056396, -1.2156344652175903, -0.0524737611413002, -0.09967637062072754, 1.3884973526000977, 0.4682559072971344, -0.6451370716094971, -0.1676330268383026, -2.392955780029297, 0.6677260398864746, 0.1700417399406433, -0.8424677848815918, -0.7307422757148743, -1.0761101245880127, -0.37715575098991394, -1.0157498121261597, -1.4148879051208496, 0.335737943649292, -0.1400945782661438, 1.3845887184143066, 0.04196573793888092, -1.3971604108810425, -0.3817060589790344, -0.8448947072029114], [0.5554342865943909, -0.8068597316741943, -0.5160515904426575, 0.5942420363426208, -0.7019934058189392, -0.19479438662528992, 1.2522969245910645, 0.36550676822662354, 0.6525781154632568, -0.42692282795906067, -1.657668113708496, 1.7808568477630615, -0.888080894947052, 0.17495238780975342, 0.5978380441665649, 1.376103401184082, -0.6456934809684753, 0.1104326918721199, -2.820871114730835, 0.9977050423622131, 0.21649053692817688, 1.2418760061264038, -1.5065079927444458, 0.54496830701828, 0.8484724760055542, 0.16563807427883148, -0.8904263973236084, -0.8945937156677246, 0.8925453424453735, -0.03946126997470856, -0.15594063699245453, 0.6818313002586365, -1.1253074407577515, 0.9917252659797668, 0.9269959926605225, -0.7641125917434692, 0.6078054308891296, 1.9558939933776855, -0.6176097393035889, 0.49557816982269287, 0.038032665848731995, 0.44993552565574646, -1.929822564125061, 1.26396906375885, 0.6239370703697205, -0.8229089379310608, -0.1114228367805481, -0.760627031326294, -0.83723384141922, -1.821590542793274, 1.3106085062026978, -0.8736895322799683, 0.657602846622467, -0.7133800983428955, 0.5571033954620361, -1.4118187427520752, 1.3450391292572021, -0.3877636790275574, 1.0120482444763184, 1.926449179649353, 0.05594348534941673, 0.3424561619758606, 1.1371456384658813, -2.594846487045288], [0.47096511721611023, 1.3226412534713745, 0.19074001908302307, -1.3008146286010742, 1.0475983619689941, 1.027880072593689, 1.0983953475952148, -0.01587699167430401, -0.020407432690262794, 0.1907886266708374, -0.9004887342453003, -0.44398683309555054, -1.1773582696914673, 2.1389412879943848, -0.9334637522697449, -0.30957862734794617, 0.7317975759506226, -0.14927184581756592, -1.1820110082626343, 0.5906558036804199, -1.1817834377288818, -1.1836633682250977, -0.6648406386375427, 0.010601609013974667, -0.19735011458396912, 0.45339322090148926, -1.442341923713684, -0.18197456002235413, 0.7346146702766418, 0.6690573692321777, -0.8464768528938293, -0.03222019970417023, -2.329785108566284, 0.9363049268722534, 0.8462883234024048, 0.9054509401321411, 2.398632526397705, -0.6374407410621643, 0.29159820079803467, 1.4204318523406982, 1.9840351343154907, -0.3708465099334717, -1.3858014345169067, 0.07632385194301605, 0.8215847015380859, -0.5492836236953735, -0.5631285309791565, -1.2133044004440308, -2.5692427158355713, -0.9961087107658386, 0.51873779296875, 1.136979341506958, 0.4094366729259491, -0.019821831956505775, 1.137013554573059, 1.0844746828079224, 1.4338672161102295, -0.6380432844161987, 0.5604246854782104, -0.29645612835884094, -2.074544668197632, -0.2467430680990219, 0.6534289717674255, 0.33322563767433167], [-0.021955592557787895, 0.2339768409729004, -0.4259001910686493, 1.222138524055481, -0.20937959849834442, 0.34858477115631104, 0.5674624443054199, -0.9489673972129822, -0.308504581451416, 0.7678006887435913, -0.21910202503204346, -1.388549566268921, 0.08449771255254745, 0.6186257600784302, -0.4415939152240753, -0.763798177242279, 0.6660840511322021, 0.645968496799469, -1.0936089754104614, -0.22737836837768555, 0.20258130133152008, -1.1871280670166016, 1.1990829706192017, -0.8519478440284729, -1.6557040214538574, 0.05032290145754814, -1.336447834968567, 0.2840350866317749, -0.1599188596010208, 0.9636701941490173, 0.2347317636013031, 0.23230555653572083, 0.6012028455734253, -1.6251304149627686, 0.3976265788078308, -0.8171606659889221, -1.4577218294143677, 0.20062138140201569, -0.44193968176841736, 1.0051064491271973, 0.8509167432785034, 0.5187353491783142, -0.341263085603714, -1.1309880018234253, -1.0618617534637451, -0.6919251084327698, -1.3650791645050049, 0.39175066351890564, -0.8941652774810791, 0.8028221130371094, -0.15034762024879456, 0.7111386060714722, -0.0238630548119545, -0.2490847408771515, -0.42350175976753235, 0.522964358329773, -0.36746856570243835, -0.5163558125495911, 0.13467369973659515, -0.07425010949373245, 1.3540899753570557, 0.10937002301216125, -0.7712393999099731, -1.4748047590255737], [-1.0002301931381226, 0.6778748631477356, 0.58184415102005, 0.2803175449371338, -0.2469150424003601, 0.018399665132164955, 0.6774027347564697, 0.9100720882415771, -0.9427134990692139, -0.6632291674613953, -0.2982042133808136, 1.0749975442886353, 1.7834168672561646, -1.408873438835144, 0.31878283619880676, -0.4390455484390259, 1.6977956295013428, -1.0164848566055298, -1.068669319152832, -0.8052303791046143, 0.8590851426124573, 0.1901574730873108, 0.3766143023967743, 0.059489887207746506, 0.9141091108322144, 0.3085559904575348, 1.0986100435256958, 0.500749409198761, 0.16535669565200806, 0.0529317781329155, -1.450943946838379, 2.54099702835083, 1.0763226747512817, 1.0686041116714478, 0.3254357874393463, 0.9530085325241089, 0.32634779810905457, -0.9762446284294128, -0.9001238346099854, 1.0203927755355835, 0.8563060760498047, 1.1207244396209717, 0.7654998898506165, 1.7518867254257202, 0.2245408296585083, 0.5574455857276917, -0.3278140723705292, -2.623904228210449, -1.312856912612915, 0.9926464557647705, 0.30087852478027344, -0.7297325134277344, -1.0761910676956177, 0.0769042819738388, -0.3727877140045166, -1.6156277656555176, -0.8692403435707092, -0.8191598653793335, 0.22691839933395386, 0.9802839756011963, -0.5279707312583923, 0.6186654567718506, 0.33814260363578796, 1.5015641450881958], [1.1618354320526123, 0.493988960981369, 1.3844859600067139, 0.7414339184761047, -0.9080285429954529, -0.6761083602905273, -0.5575858950614929, -0.6833892464637756, -0.8114391565322876, -0.4308142364025116, 0.16121399402618408, 2.500312089920044, 0.8137304186820984, -1.7381296157836914, 0.4026550054550171, 0.8773110508918762, 0.8633256554603577, -0.3513473570346832, -0.9255152344703674, -0.5280545353889465, 0.6892404556274414, -1.4289671182632446, 1.2306684255599976, 1.2771695852279663, 0.24383459985256195, -0.8262377977371216, 1.3158999681472778, 1.0098246335983276, 1.6209319829940796, -1.8020838499069214, -1.7990217208862305, -0.25354039669036865, 0.7428227663040161, -0.8615741729736328, -0.07871001213788986, -0.8631320595741272, -1.0491135120391846, -0.8539043068885803, 0.7936334609985352, -2.545877695083618, -0.5614486932754517, -1.127042531967163, -1.0958688259124756, 0.8951970934867859, -0.6411380171775818, 0.6227720975875854, -0.12756718695163727, -0.34730780124664307, -0.8119890093803406, 0.7318814396858215, 0.23109984397888184, -1.008239507675171, -0.21778623759746552, -0.18192927539348602, -1.575668454170227, 0.4050636291503906, -0.4999028444290161, 0.6457343697547913, 2.4648425579071045, 0.7209489941596985, 0.370286226272583, -1.0983288288116455, -0.5825972557067871, 0.6987891793251038], [-1.2121690511703491, -0.7086141109466553, 0.3585400879383087, -0.318498432636261, 1.765479326248169, 0.24055930972099304, 0.880040168762207, 1.0528438091278076, 0.45281144976615906, -1.5997157096862793, 0.4393913447856903, 0.13312552869319916, -0.02470420114696026, 2.3047432899475098, -0.5428563952445984, 1.3802473545074463, 1.1700477600097656, 1.248348355293274, -0.6274263262748718, -0.15346497297286987, -1.2681649923324585, -0.44335028529167175, -0.5095714330673218, -1.212151288986206, -0.014477148652076721, -2.8361783027648926, -0.6363583207130432, 0.4859991669654846, -0.9195978045463562, 0.4081951677799225, 0.6667873859405518, -0.6524428725242615, -1.323925256729126, 1.565028429031372, 0.7911509871482849, 1.139137625694275, 0.30776548385620117, -0.2726023495197296, 1.3058608770370483, -0.17960159480571747, 0.5552458167076111, -0.8575539588928223, -1.3161709308624268, 2.17781138420105, 1.7783703804016113, 0.6721544861793518, 0.030796166509389877, -0.6000433564186096, -0.9844650626182556, 1.2017558813095093, 1.1334325075149536, -0.12421587109565735, -0.08306890726089478, -0.13659417629241943, 1.3033643960952759, -0.1080002412199974, 0.652644693851471, 1.0736525058746338, 0.5761659145355225, 1.0117067098617554, 0.3283841013908386, 0.7249724864959717, -0.19870010018348694, 1.11549711227417], [0.19435414671897888, -0.22441616654396057, 0.4776882529258728, -1.9720072746276855, 2.4553451538085938, -0.02514241822063923, 1.3882980346679688, 1.3058172464370728, -0.6177647113800049, -0.13077539205551147, -0.350880891084671, -0.12191921472549438, -0.746803343296051, 1.8048436641693115, 0.10584747791290283, -1.3810962438583374, -1.683234453201294, -0.6249651312828064, -1.2631133794784546, -0.5209771990776062, -1.4941715002059937, -2.3180854320526123, -0.21091198921203613, 1.2984464168548584, -0.15845593810081482, -0.7919344305992126, 1.0656629800796509, 1.692514419555664, 0.5841474533081055, 0.353731632232666, 0.7110145092010498, 0.787646472454071, -1.5734654664993286, 1.0803853273391724, 0.19035400450229645, 0.13974733650684357, 0.22396442294120789, 1.0789436101913452, -0.03724118694663048, 0.045761436223983765, -0.7526825070381165, -0.8953517079353333, -1.2566903829574585, -0.6798192858695984, 0.5670883059501648, -0.2462388575077057, -1.0422383546829224, -0.13966941833496094, 0.6269130706787109, -0.9202399253845215, 1.4903740882873535, -0.1446840912103653, 0.08942698687314987, -0.7976913452148438, 0.5446481704711914, 1.5146359205245972, 0.9469380378723145, -1.3689312934875488, -1.5915430784225464, 1.1923984289169312, 0.31835293769836426, 0.5067513585090637, 0.48560982942581177, 0.35642209649086], [-1.075945496559143, -0.8408649563789368, 0.4995635449886322, 0.8887091279029846, -0.6182632446289062, -1.9978015422821045, -1.1590615510940552, -0.9145305156707764, -1.8170164823532104, 0.6367102861404419, 0.035068780183792114, -0.10055415332317352, -2.1059601306915283, -1.4279011487960815, -0.06358896940946579, -2.4663078784942627, 0.10522940754890442, -1.4170212745666504, 0.9700515866279602, -0.47737324237823486, 0.11696738749742508, 1.2018096446990967, -0.10718511044979095, 1.258530855178833, -2.244468927383423, -1.3484375476837158, -0.8937314748764038, 0.6158507466316223, -0.15433885157108307, -0.7967312335968018, 0.11737757176160812, 0.8670182228088379, -0.6396586894989014, 0.3028016686439514, 2.0224874019622803, 1.6838505268096924, -0.8548743724822998, -0.10980895161628723, -1.173912763595581, -0.41244328022003174, -0.6091004014015198, 0.1410447210073471, -0.9808842539787292, 0.411827027797699, 0.0429324209690094, -0.6521705389022827, -0.2621729373931885, 1.0932414531707764, 0.28711044788360596, 2.0159120559692383, -0.276947021484375, 0.22834599018096924, -1.8943508863449097, -0.8818051815032959, -1.798744559288025, 0.7977569699287415, 0.19276611506938934, 0.2929600477218628, -0.36143264174461365, -1.1020194292068481, 1.0353387594223022, 0.5171114206314087, 1.4742223024368286, -0.4075736105442047], [-0.1778571456670761, 1.3044790029525757, -0.027664532884955406, -1.6769853830337524, 0.5804199576377869, -0.04482405260205269, -1.4917787313461304, -0.7341581583023071, 1.218570590019226, 1.0649269819259644, -1.0451511144638062, 1.098983645439148, -0.6032511591911316, -0.5070504546165466, -1.3645148277282715, -1.0753618478775024, 0.046602196991443634, 0.39430370926856995, -0.7248289585113525, 3.834709405899048, -0.9858259558677673, 0.542789876461029, 0.5393344759941101, -0.7688992023468018, 1.114060401916504, 1.5352195501327515, -1.5357199907302856, -0.15938423573970795, -0.7626888155937195, -0.4300991892814636, -1.3645012378692627, 0.9217755794525146, 1.4290945529937744, -0.20049406588077545, 1.0743980407714844, 0.8239554762840271, 1.1829880475997925, 0.1098078116774559, 0.21925070881843567, -2.609938144683838, 1.654982089996338, 0.13316801190376282, 0.2808727025985718, -0.5038378238677979, -0.9476848244667053, -0.0031136993784457445, 0.8436757922172546, -1.178061842918396, 0.09740804880857468, 0.37286555767059326, -0.8970929980278015, -0.6932978630065918, -1.1137306690216064, -0.7680066823959351, -0.029986314475536346, -0.017853721976280212, 0.6908767223358154, 0.057189133018255234, -0.9167511463165283, -1.0385388135910034, -1.372725486755371, 0.3731096088886261, 0.33792194724082947, 0.22888539731502533], [-0.9955950379371643, -2.589114189147949, 0.26132306456565857, 0.5606557726860046, 1.1390219926834106, -1.7493526935577393, -0.17776410281658173, -0.2922894060611725, -0.7887979745864868, -0.28631946444511414, 0.9536950588226318, -1.4182226657867432, 1.284277319908142, -2.0727615356445312, -0.30984213948249817, 0.3690531551837921, -0.6911920309066772, -0.834821343421936, -0.7494384050369263, 0.13555093109607697, -2.697082281112671, 1.094892978668213, 0.32131439447402954, -0.11172253638505936, 1.06027352809906, -0.7938839197158813, -1.6738457679748535, 1.37330162525177, 1.2889443635940552, 1.3192040920257568, 0.45521363615989685, -1.0215301513671875, 0.47532302141189575, -0.6203112602233887, 1.0374575853347778, -0.2631681263446808, 0.6915573477745056, 0.8986833691596985, -0.980522632598877, -0.2582075297832489, 2.050370216369629, -0.6830343008041382, -0.268978089094162, -1.2795720100402832, 0.7416931986808777, 0.09560608118772507, -1.7249423265457153, 0.009572348557412624, -0.5936405658721924, 1.957699179649353, -1.332959771156311, 1.182310938835144, 0.3277136981487274, -1.368567705154419, 0.7854982614517212, 0.9179277420043945, -0.8571447730064392, 0.14816385507583618, -1.4039124250411987, 0.2551252245903015, 1.1899806261062622, -1.2199172973632812, 0.5189706087112427, 2.012892246246338], [1.2487159967422485, -0.38682684302330017, 1.5121983289718628, 0.7249534726142883, -0.4690599739551544, 1.5344525575637817, -0.5688627362251282, -0.5198912024497986, -0.8895718455314636, -1.3140827417373657, -0.10914167761802673, 0.9830882549285889, 1.6212469339370728, 0.22757352888584137, 2.086901903152466, -1.2782400846481323, -2.164076566696167, -0.3624712824821472, -1.5976440906524658, -0.37105897068977356, 2.2746944427490234, 1.1458786725997925, 0.759768009185791, 0.6442384123802185, 1.0783294439315796, -0.5760116577148438, 0.5331632494926453, 1.7773138284683228, -1.7768226861953735, -0.18135353922843933, 0.4663136899471283, 0.14879761636257172, -0.45576027035713196, 1.6039879322052002, 1.4789763689041138, 0.5152400732040405, 1.0887936353683472, -1.3718992471694946, 0.9099281430244446, -0.13597959280014038, 0.017241686582565308, -0.45127072930336, 0.18127691745758057, 1.667165756225586, 0.6355376839637756, -0.8309069871902466, 0.4974353313446045, -0.9144512414932251, 1.2754796743392944, 1.2819782495498657, -0.4218807518482208, -0.7251208424568176, 1.5121687650680542, -0.39578738808631897, 1.4320465326309204, -0.9109520316123962, -0.9571589231491089, -1.3033818006515503, -0.5251776576042175, -1.3051332235336304, 0.32852065563201904, 1.5404032468795776, -0.014961445704102516, 0.7962262034416199], [-0.46697285771369934, 0.27965492010116577, -0.25994738936424255, -0.6334831714630127, -2.2352709770202637, -0.36220207810401917, -1.29300057888031, 2.2680790424346924, 0.5390358567237854, -0.6873663067817688, 0.1522872895002365, -0.6129352450370789, 1.7349156141281128, -0.9139836430549622, -0.17778795957565308, 0.9262533187866211, -1.321785807609558, 0.7355216145515442, 1.1949489116668701, -0.34600627422332764, -0.5774654150009155, 0.6694685816764832, 1.093686580657959, -0.2534004747867584, 2.6135871410369873, -2.2660892009735107, -0.18401126563549042, 2.106370687484741, -0.607728123664856, -1.4289449453353882, 0.5782545208930969, 0.909423828125, 0.3814520239830017, -1.9141796827316284, 0.9436922073364258, -1.3415591716766357, 1.7187870740890503, 1.0944300889968872, 0.1831832081079483, -0.4318731427192688, 0.4818909168243408, 0.6343533992767334, 0.2114648073911667, -2.2520029544830322, -0.4497547149658203, -1.1077536344528198, -1.3705722093582153, 1.4113521575927734, 0.5353280901908875, -0.5332382917404175, -0.8715000748634338, -1.3593426942825317, -1.4515252113342285, -0.665993332862854, -0.10510946065187454, 0.9497950077056885, -1.0561480522155762, -1.0405588150024414, -2.358346939086914, 0.6008450984954834, -0.07191658020019531, -1.6212562322616577, 1.0402305126190186, 0.3072870373725891], [0.41658148169517517, -0.6232201457023621, -0.6002256274223328, 0.07816474139690399, 0.09976093471050262, -0.4200925827026367, -0.9739251136779785, -0.8482878804206848, -0.9935646057128906, 1.4858981370925903, -0.19688989222049713, 2.1767923831939697, -0.8895025849342346, 0.6742618083953857, -0.2880052328109741, 0.8613616824150085, -0.4593504071235657, -0.42897912859916687, -1.1425143480300903, 0.2048081010580063, -0.07515152543783188, -0.15051592886447906, -1.0503573417663574, 1.1478095054626465, 2.8864026069641113, -1.201786994934082, 2.2035889625549316, -0.6527217626571655, 0.5222858190536499, -0.8978546261787415, -0.45122256875038147, -1.0221221446990967, 0.6794273257255554, -1.703348994255066, 0.8508414030075073, 1.2062962055206299, 0.3693041503429413, 0.7186859846115112, -3.3260364532470703, 0.007169095333665609, 0.24401354789733887, -0.6117013692855835, 0.24175122380256653, 0.07191399484872818, -1.1012850999832153, -2.4467923641204834, -2.1947214603424072, -0.8127919435501099, 0.2681616246700287, 0.08629845827817917, 0.3335411250591278, -0.634634792804718, -1.9146567583084106, -0.17733295261859894, -0.4311358630657196, 0.9075034856796265, -1.6586753129959106, 0.5920937657356262, 0.8472951054573059, -0.21468907594680786, -0.6348043084144592, -0.21088166534900665, -0.6886764168739319, 0.23632560670375824], [-1.108597755432129, -0.885799765586853, 0.4448966085910797, 0.2867518663406372, -2.516939878463745, 0.07880334556102753, -0.5373640656471252, 0.2547571361064911, -0.2639386057853699, 0.012639559805393219, -0.3090626299381256, -2.251589059829712, -0.21164833009243011, -1.396329641342163, 0.33851221203804016, -0.8166630268096924, 1.5665639638900757, 1.5496193170547485, 0.06592021882534027, -0.8020036816596985, 0.5747286081314087, -1.2154940366744995, -0.16244620084762573, 0.5649512410163879, 0.8747410774230957, 1.5693069696426392, -0.7099413275718689, 1.6926881074905396, 0.6111883521080017, 0.732969343662262, 0.3767209053039551, 0.5561541318893433, -0.7062957286834717, 0.5013231635093689, 1.8081756830215454, -1.0010367631912231, -1.6678909063339233, 1.8272639513015747, -1.4905370473861694, -0.19357995688915253, 1.0336955785751343, 0.3304646611213684, -0.873461902141571, -1.2943543195724487, 0.6752851605415344, -0.7589446902275085, -0.15795831382274628, 1.1293176412582397, 0.4042206406593323, 2.2961785793304443, -1.3956685066223145, -0.4141097664833069, -1.1769410371780396, -0.43802329897880554, -0.4416065216064453, -0.2571031153202057, -1.1788313388824463, 0.8952701687812805, 2.7561943531036377, -0.2330397069454193, -2.3346855640411377, -0.8095805644989014, -0.08860665559768677, 1.0053906440734863], [0.3477693796157837, 1.3423784971237183, -0.5898228883743286, -0.4280944764614105, 0.6908577680587769, -0.39058974385261536, -2.6301209926605225, 0.27958935499191284, -0.5384750366210938, -0.2568994164466858, -0.9494255185127258, 2.338550567626953, -0.06189917400479317, -2.245204448699951, -0.3140415847301483, 0.8033521175384521, -1.457506775856018, -0.17263922095298767, 0.7245713472366333, 0.3358543813228607, -1.6599256992340088, -0.0006714219343848526, -1.5536361932754517, 0.23774050176143646, 0.8777776956558228, 1.1568058729171753, 1.5583845376968384, 0.397182822227478, -1.448314905166626, -1.1036336421966553, -0.5575016140937805, -1.0577083826065063, -0.8643127679824829, 1.644889235496521, 0.16615255177021027, -1.518456220626831, -1.3408117294311523, 0.44026848673820496, -0.46366676688194275, 0.9307281374931335, 0.9677506685256958, -2.1294472217559814, 0.47080880403518677, -0.7396473288536072, -1.217282772064209, 0.0685470849275589, -0.7917225360870361, 0.7058118581771851, -1.3268680572509766, -0.7877678275108337, -0.042827997356653214, -1.3439558744430542, -0.577728807926178, -0.7942899465560913, -0.11943810433149338, -1.4137821197509766, 1.8623489141464233, 1.4991616010665894, 0.6208921074867249, -0.7749141454696655, -0.21617674827575684, 1.0534980297088623, 0.25322332978248596, -0.008649824187159538], [0.12451756000518799, -1.6509525775909424, -0.26271867752075195, 2.1063003540039062, -0.32871055603027344, -0.19641916453838348, -1.0590448379516602, 0.8065124750137329, 0.8718850016593933, -0.15728910267353058, 1.5767693519592285, 1.2918963432312012, 1.098570704460144, -0.6861182451248169, -0.7643784880638123, -0.06670809537172318, -0.8627520799636841, -0.2754586935043335, 0.35562416911125183, -1.4474235773086548, -0.36417049169540405, 0.26313817501068115, 0.6502074003219604, 2.6159772872924805, -0.04307728260755539, 0.8638482689857483, 0.07793249189853668, 1.0568733215332031, -1.3844548463821411, -2.7835373878479004, -1.150050163269043, -1.0727976560592651, 1.8723171949386597, -0.07355774939060211, 0.7854729890823364, 1.590017318725586, -0.18570296466350555, 0.7129083871841431, -1.4661165475845337, 0.4202885627746582, -1.1480967998504639, -1.7230271100997925, 0.9892264604568481, 0.8186166882514954, -0.19061322510242462, 1.555416464805603, -0.20795640349388123, 0.8404645323753357, -1.5352755784988403, 0.08548379689455032, -2.8424055576324463, -0.40439528226852417, 0.28942790627479553, -0.4798680245876312, -1.176928162574768, 0.7447832226753235, -0.32772910594940186, 0.9783983826637268, -0.3719603717327118, -0.09198091924190521, -1.7942676544189453, -1.2616121768951416, -0.778520405292511, 1.923865556716919], [-0.6638283133506775, -0.23246634006500244, -1.8490968942642212, -1.9338124990463257, 0.26553595066070557, -0.122193843126297, 0.6853318214416504, 2.1913692951202393, -1.1448943614959717, -1.0137465000152588, 0.97347491979599, -1.7658401727676392, -1.6313185691833496, -0.3412921726703644, -0.30187681317329407, 0.19343964755535126, 0.20523642003536224, -0.5631827712059021, -1.323896884918213, -2.2921063899993896, 0.06635086238384247, -0.18891355395317078, -2.6872670650482178, 0.4516340494155884, -0.9491585493087769, -0.6822054386138916, -0.05018240585923195, -0.7558555603027344, 0.7999964952468872, 0.27295824885368347, 0.653313398361206, 1.9157986640930176, 0.3451041281223297, -0.6419153213500977, 0.8628770709037781, -0.8309107422828674, -2.952505350112915, 0.743628978729248, 0.5597797632217407, -1.1991633176803589, 0.9401987195014954, 1.8480536937713623, 0.5065088272094727, -0.06284290552139282, 0.08514559268951416, -0.7518012523651123, 0.4672945439815521, 0.6441845893859863, 1.0989903211593628, 0.008431484922766685, 0.8466165661811829, -0.9723979830741882, -0.6071648001670837, -0.2841186225414276, 0.9098027348518372, 1.6719996929168701, 0.7992660999298096, -1.1280733346939087, 0.028437698259949684, 0.87940913438797, -0.3313637971878052, 1.8119568824768066, -1.178390383720398, -1.2067090272903442], [-0.1197362169623375, -1.2500354051589966, 0.5811348557472229, -1.7090202569961548, 1.491383671760559, -0.020450761541724205, 1.0759092569351196, -0.5151193141937256, 0.015834404155611992, -2.3458304405212402, -2.029792547225952, -0.3623765707015991, -1.0806233882904053, -1.706259846687317, -0.730236291885376, -0.4190022051334381, 0.593527615070343, 0.778338611125946, -1.746010661125183, 0.2505556643009186, 0.15614783763885498, -0.6043074131011963, -0.4145328998565674, 0.369329571723938, -1.4833130836486816, -0.31609880924224854, 0.8106671571731567, 2.301679849624634, 1.29212486743927, 0.7682818174362183, -2.1589035987854004, -0.26620516180992126, -1.4523905515670776, 2.251237630844116, 0.1222912073135376, -1.4079084396362305, -1.936023473739624, 0.01116381399333477, -0.37986549735069275, -1.5331544876098633, 1.1958409547805786, 0.3500571846961975, -0.9459115266799927, -0.011350017972290516, 2.4273860454559326, -1.0191811323165894, -0.4487311840057373, -1.8085335493087769, -0.9449382424354553, -1.2726401090621948, 1.3656713962554932, -0.6672436594963074, 0.4198586046695709, 0.4624183773994446, -0.6358945369720459, 0.9011989235877991, -0.08176244050264359, -0.8658562302589417, 0.2151746153831482, -1.3182868957519531, -0.3868866264820099, -0.19346728920936584, 0.6175569891929626, 0.11683794856071472], [-1.164637804031372, -0.6253565549850464, 0.1346852034330368, -0.28452250361442566, -0.004064058884978294, 0.3994336724281311, -1.7521823644638062, 0.3644990921020508, -0.6673902273178101, -0.2648918330669403, -0.49647045135498047, -0.5876618027687073, 0.04881511628627777, -0.6454327702522278, -1.6008228063583374, -0.28256985545158386, 0.031109681352972984, -0.6401470303535461, 0.016470789909362793, -1.1760700941085815, -0.8955557346343994, 1.8050585985183716, -0.25691869854927063, -1.154992938041687, -0.5014397501945496, -0.26304060220718384, -0.41035398840904236, -1.5447766780853271, 0.13257867097854614, 0.6291429400444031, 0.35251954197883606, -1.9348700046539307, -0.3282528519630432, -0.787696361541748, -0.01255748514086008, -0.14932195842266083, 0.28559839725494385, -0.008690758608281612, -0.8166991472244263, -0.547045111656189, -0.24535585939884186, 0.4845406711101532, -2.300961494445801, -0.14831124246120453, -1.5604122877120972, -1.4154435396194458, -0.7752512693405151, 0.3299091160297394, 0.6109015345573425, -1.9295389652252197, -0.5380843281745911, 0.9280198216438293, -0.39571136236190796, -1.0663379430770874, 1.1434712409973145, 1.2170308828353882, 0.36001190543174744, 0.8064438700675964, -0.0782892033457756, 2.5445549488067627, -0.6723807454109192, -0.36835357546806335, -1.410308837890625, 1.5448213815689087], [0.18158748745918274, 0.30032360553741455, -0.7270719408988953, -1.1972547769546509, -0.3268533945083618, 2.0976247787475586, 0.006329705473035574, -0.13420113921165466, -0.0007309917127713561, -0.14267900586128235, 0.9659155011177063, -0.4644286036491394, 0.08388526737689972, -1.5794588327407837, -0.5459269285202026, 0.40912145376205444, 0.8714861273765564, -1.7548578977584839, 2.0474467277526855, 0.5138066411018372, 0.7879856824874878, 0.02160036377608776, 0.6193763613700867, 0.5391960144042969, -0.6544656157493591, 0.5632607340812683, 0.001445890637114644, -2.083613872528076, -1.028296947479248, -0.2568303048610687, -1.2004544734954834, 0.2018868327140808, 0.3987463116645813, -0.34224623441696167, -0.1071120873093605, 0.9142850041389465, 2.6367108821868896, 0.06870350241661072, 0.1307476907968521, -0.7648055553436279, -1.1062477827072144, -0.7477232217788696, -0.09793435782194138, 0.5373862385749817, -0.9407786130905151, -0.1039859876036644, 1.2010642290115356, 1.2472718954086304, 0.1337042897939682, 1.1570278406143188, -1.101539969444275, -1.952638864517212, -0.45787885785102844, 1.441542625427246, 0.5772119164466858, -0.09982946515083313, -0.9268798828125, -0.8961565494537354, 0.09995225816965103, 2.3089442253112793, -1.0239150524139404, -0.4789804518222809, -1.0639129877090454, 0.07821953296661377], [1.3454352617263794, -1.087247371673584, -0.22813822329044342, -0.024342060089111328, 0.6407760381698608, 0.2667991518974304, 1.412157654762268, 1.2164477109909058, -0.29271838068962097, 0.41688352823257446, 0.2303149402141571, 0.6119796633720398, 0.014065033756196499, 1.2781703472137451, -0.17438212037086487, 1.4134769439697266, -1.6052865982055664, -0.18825815618038177, -1.7642240524291992, -0.8360713124275208, -0.8919859528541565, -0.368056982755661, -0.29593920707702637, 0.3734351694583893, -0.1896502822637558, -1.8159507513046265, 0.11900458484888077, 1.337834358215332, -0.2241055816411972, -0.6843923330307007, 0.34958359599113464, 0.20276246964931488, -1.3487051725387573, 0.41495388746261597, 0.7395660877227783, -0.8723563551902771, 0.21382759511470795, 1.036281704902649, 3.084395408630371, -0.48927584290504456, 0.36994871497154236, 1.2699095010757446, -0.050570741295814514, 0.6302018761634827, 1.3553203344345093, 1.0943259000778198, 1.7241389751434326, 0.404954731464386, -0.3893994688987732, -0.25643959641456604, -0.8543850779533386, 1.0343554019927979, -0.5097631812095642, -0.06403303891420364, -0.2185784876346588, 0.0026065742131322622, -0.07204735279083252, 1.3298838138580322, 0.8530899882316589, 0.336448073387146, -0.7509865760803223, 1.4195903539657593, 1.0644642114639282, 0.8708493709564209], [0.027805963531136513, 0.8062717914581299, 0.8696243762969971, 0.008609460666775703, 0.03591523692011833, 0.9963822960853577, -1.3548702001571655, -0.4156954288482666, 0.3589695394039154, -0.6818220615386963, 0.5988208651542664, 0.8935259580612183, 1.230299472808838, -0.6099364757537842, 1.5193233489990234, 1.01570725440979, -2.4988973140716553, -1.0596009492874146, -0.6073216795921326, 0.40520718693733215, -1.74008047580719, -0.07150368392467499, -1.8542605638504028, 0.20558607578277588, 0.3343481421470642, -1.7654237747192383, 0.1872786283493042, 1.7070183753967285, -1.3198822736740112, -1.9796178340911865, -0.9448503255844116, 0.8661790490150452, -0.29922181367874146, -0.7040921449661255, 0.381064772605896, -0.6562834978103638, 0.8083892464637756, 1.597675085067749, -1.6340149641036987, 0.028091885149478912, 2.8332760334014893, -0.9500271081924438, -0.36065953969955444, 0.7158632874488831, -0.4982813894748688, -1.518784523010254, 1.4600685834884644, 0.29741808772087097, 1.2356420755386353, -0.5286920666694641, 0.233856201171875, -0.007320417556911707, 0.5276665091514587, -0.8066348433494568, 1.5364614725112915, 0.4316859543323517, -0.5307353734970093, -0.6020424365997314, 1.3662378787994385, -1.1486170291900635, 0.3687463104724884, 1.2434006929397583, 0.4240083396434784, -0.7102631330490112], [-2.773815393447876, -0.14720158278942108, -0.41039973497390747, 0.2318364381790161, 0.0965098962187767, -1.766619324684143, -1.3340519666671753, 0.04057525843381882, 1.9349159002304077, -0.257342129945755, -0.6494002342224121, 0.08760391175746918, 0.7413060069084167, 1.4901659488677979, -0.4326852560043335, -1.3658852577209473, -2.5162408351898193, -0.3492092788219452, 0.8017280101776123, 1.7621904611587524, 0.09186327457427979, -1.4438514709472656, -1.183129906654358, 0.2774219810962677, 1.0835161209106445, -0.3450509011745453, 0.4811199903488159, 0.6530094742774963, -1.00680673122406, 0.2723953127861023, 0.10078594833612442, 1.562727689743042, -0.31979578733444214, -0.6523028016090393, 0.5751008987426758, 0.9715662002563477, -0.4082637131214142, -0.5792626142501831, 0.25622400641441345, 0.9891210198402405, -1.2344733476638794, -0.28496843576431274, -0.47706353664398193, 0.3467652499675751, 0.9645105004310608, 0.24819058179855347, 0.16673213243484497, 0.04817589372396469, -0.0025513279251754284, 0.480223149061203, 0.4483487904071808, -1.4771367311477661, -0.6624323129653931, 1.6078143119812012, -0.5873028039932251, 0.8377411365509033, 0.6227692365646362, -0.40732628107070923, 1.2233630418777466, -0.6106622219085693, 0.8627969026565552, 1.315779447555542, -0.07264116406440735, 1.2279025316238403], [2.2916862964630127, -0.7599590420722961, 0.4906027019023895, 0.6119392514228821, 1.8291776180267334, -0.9470319151878357, -0.6685928702354431, -0.4326358735561371, 1.0512678623199463, -0.45336854457855225, -0.20111359655857086, -1.1128531694412231, -0.009308888576924801, -0.3093290328979492, -0.40742576122283936, -2.0281245708465576, -0.758183479309082, 2.0025041103363037, 0.4591902494430542, 0.7124625444412231, -1.8303475379943848, 0.2071210891008377, -0.46823444962501526, -0.18080587685108185, 0.8534212112426758, -0.4795382022857666, 0.8187390565872192, 1.0514572858810425, 1.3704447746276855, 0.5032828450202942, 0.586496889591217, 1.8772507905960083, -1.064592957496643, -0.49737274646759033, -0.1500888466835022, -0.23993712663650513, -0.26663148403167725, -0.9856601357460022, 0.8226187229156494, 0.3715756833553314, 0.2590772211551666, -0.9096136689186096, -0.5059481859207153, -1.1735484600067139, 0.028034456074237823, -0.843658447265625, 1.4502497911453247, 1.7037373781204224, -0.35376641154289246, 0.4556569457054138, 0.45134055614471436, 0.08769257366657257, -0.04416084662079811, -0.5715113878250122, 1.0681833028793335, -0.5560846328735352, -0.12411372363567352, 2.362788200378418, 1.5315301418304443, 0.33687877655029297, 0.08427538722753525, -1.1979974508285522, -1.3008136749267578, -1.1420258283615112], [1.1674141883850098, 0.2865769565105438, 0.39112550020217896, 1.0223547220230103, 1.6470016241073608, 0.29188698530197144, -1.0304497480392456, 0.25972694158554077, -0.9000498056411743, -0.3922153115272522, -1.225899338722229, -0.99951171875, -1.6134730577468872, -0.79039466381073, -1.5415581464767456, 1.5682940483093262, -0.20411115884780884, -0.2988348603248596, -0.30062952637672424, 0.6688100099563599, 1.0260530710220337, 1.9279389381408691, -1.2264049053192139, 1.8299813270568848, 1.637292504310608, -0.4792015254497528, 1.764549970626831, -0.10648828744888306, -0.9576772451400757, -0.22312620282173157, 0.25211355090141296, -0.13053834438323975, -3.4961180686950684, 0.09034628421068192, 0.25338080525398254, 0.9648486971855164, -0.13290804624557495, -1.1056056022644043, 0.5707619190216064, -0.7338029146194458, 0.5850000977516174, -1.5175023078918457, -0.7413814663887024, 0.007617324125021696, 1.483099341392517, 0.19024604558944702, -0.33118465542793274, -0.9630721807479858, -0.011646131053566933, 0.21417906880378723, 0.8241207599639893, -1.0340644121170044, 0.6861026287078857, -0.620389997959137, -0.8366904854774475, 0.7678509950637817, 0.4559040665626526, 1.969931960105896, 0.21544693410396576, 0.23448476195335388, -2.292308807373047, -0.36591699719429016, 0.24273885786533356, -1.1313605308532715], [0.45030075311660767, 0.3128611147403717, 0.5605894923210144, -0.8247496485710144, 0.27846863865852356, -0.7019705772399902, 1.498274326324463, -0.9310542941093445, 1.7150510549545288, 0.24836000800132751, -0.5277841091156006, 2.2998621463775635, -0.7460544109344482, 1.2749145030975342, -0.44219350814819336, 1.4051446914672852, -2.223566770553589, -0.22295241057872772, 0.6596823334693909, 0.7000753879547119, -0.7467213273048401, 0.33603647351264954, 0.01918916590511799, -0.5252830982208252, -0.13616614043712616, -1.3211627006530762, -0.2552162706851959, 0.8124535083770752, -0.11871378868818283, -0.6116759181022644, -0.9220231771469116, -0.3438586890697479, 0.7476108074188232, 0.963794469833374, -0.7905071973800659, -1.4316697120666504, 0.1949770301580429, -1.2025229930877686, 0.6345289945602417, -1.000612497329712, -0.3890002369880676, -0.7051914930343628, 0.06828727573156357, -0.655022144317627, 0.7455441951751709, -0.7570385932922363, -0.47421786189079285, -1.8158884048461914, 0.40387845039367676, -0.13544882833957672, -0.5556949973106384, -0.33450379967689514, 2.0114331245422363, -0.7700968980789185, -1.9482303857803345, -0.5113726854324341, 0.2749602794647217, 0.4505157470703125, -0.9987298250198364, -1.6786198616027832, -2.1597299575805664, 0.9753240942955017, -0.16872599720954895, 0.08429557085037231], [-1.3706406354904175, -0.06629540771245956, -0.2638178765773773, 1.657920002937317, -0.707231879234314, 0.3017042875289917, -0.46141183376312256, 0.396696537733078, -0.10312367975711823, -1.3067952394485474, -1.5160112380981445, -0.8241890668869019, 0.6754586696624756, 1.1047086715698242, 0.069020576775074, 0.03193291649222374, -0.7863552570343018, 0.032553162425756454, 0.705563485622406, 0.11077616363763809, -0.7385977506637573, 0.27858489751815796, 0.09416945278644562, 1.014941930770874, 1.0567982196807861, 0.15758194029331207, -1.1250869035720825, 0.6642350554466248, -0.5399149060249329, 1.7379857301712036, 0.9880247116088867, -0.4856383204460144, 0.28662192821502686, 0.2638739049434662, -1.7681982517242432, -0.7585340738296509, 1.024706244468689, -0.02436871826648712, 1.130403757095337, 0.3740691840648651, 0.5727119445800781, 1.1032317876815796, 0.6015433073043823, 2.13999342918396, -0.1591910719871521, -1.3791865110397339, -0.26341012120246887, -0.8018876910209656, 0.7360355257987976, 1.1575833559036255, 0.5146268010139465, 1.100726842880249, 1.0287222862243652, -0.09670950472354889, -0.35851138830184937, -1.1078001260757446, 0.4590919315814972, -1.1751103401184082, -0.03167320415377617, -0.4042665362358093, 0.671985924243927, 1.2714760303497314, -1.3020820617675781, 0.584262490272522], [0.16638925671577454, -1.742485523223877, -0.19435696303844452, -0.6597372889518738, -2.0977180004119873, 3.2799530029296875, 1.3314682245254517, -0.6192619800567627, -0.01116064190864563, -0.4312083423137665, -0.5855751037597656, -0.9697437286376953, -1.19912588596344, -0.3863750398159027, -0.9214804768562317, 1.63375985622406, -0.12855760753154755, 0.7827033996582031, 0.7803831100463867, 1.0162568092346191, 1.4562524557113647, 1.6086742877960205, 1.0020567178726196, 1.2347056865692139, -0.25842487812042236, 0.418366014957428, -0.5888444781303406, 0.250145822763443, 1.6561644077301025, -0.034493882209062576, -1.3309400081634521, 1.9476137161254883, 0.6661884784698486, -1.2896257638931274, -1.5442686080932617, -0.02550838328897953, -1.0168553590774536, 0.904371440410614, 0.5723461508750916, -0.18458135426044464, 2.1410398483276367, -2.196938991546631, 0.38036417961120605, 0.51963210105896, -0.24530364573001862, 0.6820755004882812, -0.379965603351593, 0.052752211689949036, -0.6349194645881653, -0.7656412124633789, -0.2509543299674988, -0.16121579706668854, -0.8807930946350098, 1.5039021968841553, 0.003640195121988654, -0.1771591752767563, 0.5199708342552185, -0.49728745222091675, -0.49720293283462524, -0.4878197908401489, -0.7848220467567444, 0.3927675187587738, 0.2175191044807434, -0.1457996368408203], [1.4016244411468506, -0.04367798566818237, 1.1937898397445679, -1.2264126539230347, 1.3307549953460693, -0.9108631014823914, 0.3549957573413849, 0.30461645126342773, 0.11896292865276337, 0.03118441067636013, -0.7641284465789795, 1.6899445056915283, 0.6021515727043152, -2.1842761039733887, -0.17083902657032013, 1.3228679895401, -1.8910472393035889, -0.9703307747840881, 0.36762410402297974, 0.15772108733654022, -1.4604277610778809, -0.8255196809768677, 0.2334369719028473, 1.032834529876709, -1.036220908164978, 2.5303523540496826, 0.7514961957931519, -0.552912175655365, -0.15610460937023163, 0.12180637568235397, -0.1246480718255043, 0.20412340760231018, 0.40811461210250854, -0.47322195768356323, 0.9871371984481812, -0.40133926272392273, -0.3276851773262024, 0.7325190305709839, 0.6866251230239868, 0.472901850938797, 2.29874324798584, 0.7828795313835144, 0.8856074810028076, -0.2743115723133087, -1.8301875591278076, -0.6496202349662781, 0.8752301335334778, -0.7278116941452026, 0.7579255104064941, -0.5672292709350586, -0.40647682547569275, 0.5703198313713074, -0.32297685742378235, 1.8655343055725098, -0.5634534358978271, -0.6885448694229126, -0.8699434995651245, -0.296613872051239, -0.19995218515396118, 0.039637863636016846, -0.15034212172031403, -1.1109888553619385, 0.9999961256980896, 1.3058726787567139], [-0.6681363582611084, 0.8606853485107422, 0.6150282025337219, 1.3436007499694824, 0.7118141055107117, 0.6059893369674683, -0.6238356232643127, 0.3587976098060608, -2.609452486038208, 2.588160991668701, 1.4721227884292603, -0.7814859747886658, -1.9075963497161865, -1.7179597616195679, 0.003250989830121398, -1.3066450357437134, -1.21177077293396, -2.2271981239318848, 0.5824468731880188, 0.04404141753911972, 0.5354565978050232, -1.6014357805252075, -1.1044453382492065, 0.008508528582751751, -1.5034849643707275, 0.3734458386898041, -0.3937055766582489, -0.6491544842720032, -1.2107723951339722, 1.036271095275879, 1.3611739873886108, 0.8538968563079834, 1.1734046936035156, -1.2968089580535889, 2.2330586910247803, 2.235896587371826, 1.2626245021820068, -0.6197237372398376, 1.778222918510437, -0.6127306818962097, -0.31611302495002747, 0.6504807472229004, -1.62312912940979, 2.1112823486328125, -0.6018673181533813, -0.390838623046875, -0.3439733386039734, 1.1645761728286743, 2.9415366649627686, 0.9211835861206055, 0.08684375137090683, 1.3875186443328857, -0.3073860704898834, -0.5058428049087524, -1.537175178527832, -0.41631442308425903, -0.8228158950805664, -1.8143380880355835, -0.30925416946411133, -0.9481159448623657, -0.4182226359844208, 0.2749926745891571, -1.0045015811920166, 1.2376819849014282], [0.19749969244003296, 0.49969199299812317, 0.957772433757782, -0.8815332651138306, -1.821006178855896, -0.8046213388442993, 2.0704479217529297, -0.9250879287719727, 0.2167843133211136, -0.38372042775154114, -0.5386956930160522, 0.2362632155418396, -1.3991624116897583, 1.0966848134994507, -0.14702874422073364, -0.8747827410697937, -0.4001745283603668, -1.041670322418213, -0.550404965877533, 0.07800724357366562, -1.3211668729782104, -1.3045393228530884, -0.5205444097518921, 2.1560328006744385, 2.1779592037200928, -1.1864640712738037, -0.23892314732074738, -1.8307197093963623, -1.3894846439361572, 0.4308429956436157, 1.0449882745742798, -1.075324296951294, -0.40863773226737976, 0.4776780903339386, -0.5224950909614563, -0.5543418526649475, 0.05864739790558815, 0.34896034002304077, 2.1505818367004395, -1.6139938831329346, -1.4871371984481812, 0.3796347975730896, 0.8485768437385559, -0.2786281108856201, 0.5554987788200378, -0.34808987379074097, -0.2676796019077301, 0.8927844166755676, -0.8656686544418335, 0.14635755121707916, -0.1954147219657898, -0.548613429069519, -1.6764510869979858, 0.8730632066726685, -0.17632602155208588, 0.5410699844360352, -1.0810365676879883, -1.2360668182373047, 0.19198668003082275, 0.36524027585983276, 1.1918221712112427, -0.384050577878952, 1.541979432106018, 0.6410744190216064], [-0.28539368510246277, 0.974113941192627, 1.3392531871795654, -0.2576662302017212, -0.6210886836051941, 1.1911981105804443, 1.105271339416504, -0.7235001921653748, 0.04504486918449402, 1.5026336908340454, 0.2911330759525299, -0.7354354858398438, 0.1746860146522522, -1.2181997299194336, -1.7702126502990723, 0.3423599302768707, -1.291926622390747, -0.06366560608148575, 0.09167362004518509, 0.618993878364563, 0.2551576495170593, 0.5756804943084717, -0.9231850504875183, 1.4022283554077148, 0.5458681583404541, 0.1645112782716751, 1.2961063385009766, 1.1290634870529175, -0.89105623960495, -0.2689329981803894, 0.4490346610546112, -0.09357805550098419, -0.7174736261367798, 0.5113034844398499, -0.555814266204834, 0.20458824932575226, 1.1208643913269043, 0.46335941553115845, 1.911125898361206, -0.8616700172424316, -0.6657007336616516, 0.4354112148284912, 0.25331488251686096, -0.36016273498535156, -0.058981288224458694, -1.3448458909988403, 2.5011942386627197, -0.145536407828331, -0.015280565246939659, 2.8004558086395264, -1.460383415222168, -0.9337475299835205, -1.3767285346984863, -0.19216495752334595, 0.33302173018455505, -0.3316226303577423, 0.35503119230270386, 0.4373715817928314, 0.10103202611207962, 1.0144661664962769, 1.2535032033920288, 0.8359367847442627, 0.24366013705730438, 0.5030327439308167], [-0.9480170607566833, 0.008751663379371166, -0.9636178612709045, 0.0028497008606791496, 0.5203741788864136, -0.39850038290023804, -0.04782542958855629, 0.8000657558441162, -0.7260037064552307, -0.03589688614010811, 0.5375105142593384, -1.1796131134033203, 1.1433782577514648, 0.39131712913513184, 0.4407692849636078, -0.2607751786708832, 0.24347324669361115, 1.0807220935821533, -0.46341586112976074, -0.06887763738632202, -0.7498598098754883, 1.4042541980743408, -0.12201931327581406, 0.03510095179080963, -1.7743157148361206, -1.5508337020874023, -0.15716642141342163, 0.5543351769447327, -1.0740711688995361, -0.1810733526945114, -0.17910975217819214, 0.7455949783325195, -1.4941344261169434, -0.7100291848182678, -0.09315498918294907, 0.11703693866729736, -0.6636339426040649, -0.05016343668103218, -1.0201735496520996, 0.5566426515579224, 1.361998438835144, -1.2800296545028687, 0.38110116124153137, -0.48283323645591736, -0.3944609761238098, -0.7955494523048401, -0.7960796356201172, 0.5223686695098877, 1.6743923425674438, 0.015599052421748638, -0.7407523393630981, 0.026502113789319992, -0.1206890419125557, -0.27246880531311035, 1.210974931716919, 1.4835559129714966, 1.3542027473449707, 1.6493034362792969, 0.0052448236383497715, -0.4966241717338562, -0.9625416994094849, 0.27371132373809814, -1.0356017351150513, -0.9508849382400513], [-0.3086128532886505, -0.2660521864891052, -0.1357884556055069, 1.9459203481674194, -0.667197585105896, -1.0883820056915283, -1.5683492422103882, -0.527295708656311, -0.47763702273368835, -1.7242580652236938, -0.05245066061615944, -0.9042267799377441, -0.5965892672538757, 0.12760254740715027, -0.6978892087936401, -0.9072138071060181, 0.6971469521522522, 1.5603926181793213, -0.44081223011016846, 0.8550897836685181, 0.31963226199150085, 0.19310708343982697, 0.8338058590888977, -1.3398712873458862, -0.4643859565258026, -1.8777962923049927, 0.13663022220134735, -2.1581873893737793, 0.7994956374168396, 0.6374335289001465, -0.033217184245586395, -2.5790226459503174, -0.9057678580284119, -0.11245976388454437, -0.8902718424797058, 0.863442063331604, 0.4229317903518677, 0.08062094449996948, -0.0024972036480903625, 0.023285960778594017, -1.0878865718841553, -0.34980419278144836, 0.8855447173118591, -1.13690984249115, 0.7386720776557922, 0.8444551229476929, 0.6334351897239685, 0.705084502696991, -0.1249721497297287, 0.22565537691116333, -1.526414394378662, -0.0076142470352351665, -0.7092499732971191, 1.3476179838180542, -0.6585628986358643, -0.1704644411802292, -1.3072106838226318, -0.7550265192985535, 1.1200246810913086, -0.950350284576416, 0.3050013482570648, 0.49230533838272095, -1.3081039190292358, -0.47081124782562256], [1.850818157196045, 1.0260844230651855, -0.003223283216357231, 1.3836597204208374, 0.9200862050056458, 0.24306048452854156, 0.7132663130760193, 0.060180600732564926, -0.06717538833618164, -0.6320303082466125, 0.30028262734413147, -1.0610597133636475, 0.2856205403804779, -0.37902170419692993, 0.9261139035224915, 0.22117620706558228, -0.9439218044281006, 0.48252421617507935, -0.7942065000534058, -3.0035901069641113, -0.73824542760849, 1.873260736465454, 0.07050760090351105, -1.6665141582489014, 0.6616430878639221, 0.9009604454040527, 0.5216504335403442, 0.33566415309906006, -0.35710379481315613, 0.6158243417739868, -0.11112959682941437, -0.4597443640232086, 0.2777147889137268, -1.1458085775375366, -0.46705976128578186, 0.3023826777935028, -0.7697145342826843, -0.06596959382295609, 0.7280362844467163, -1.7945266962051392, -0.9835051894187927, 0.4996449053287506, 0.20642872154712677, -0.6667524576187134, -0.8209594488143921, 0.31169283390045166, -0.2819526195526123, 0.9803400039672852, -0.5192808508872986, -0.5867493748664856, -2.0608716011047363, 0.4180453419685364, -1.4565616846084595, -1.2347224950790405, 1.019445538520813, -0.22191977500915527, 1.386361002922058, 2.0973548889160156, 0.06599627435207367, -0.37824153900146484, -1.1521122455596924, -0.7362709641456604, -0.8050792217254639, -1.6842560768127441], [0.6945599317550659, -0.08798901736736298, 0.6895768642425537, 0.43694648146629333, -0.6291801929473877, -1.7277071475982666, -0.7068082094192505, -0.04072674363851547, -0.8847606182098389, -0.28853514790534973, 1.129228949546814, -0.4083894193172455, 0.183019757270813, -0.37487146258354187, 0.1888365000486374, 0.5192781090736389, 0.37450456619262695, -0.8460756540298462, -0.08192505687475204, -0.024431569501757622, -0.6506687998771667, -1.3359955549240112, 0.5596664547920227, -0.14279165863990784, 0.2681099474430084, -0.9475221633911133, -0.9468995332717896, -0.8630818128585815, -0.36092409491539, 1.561028242111206, -2.528069257736206, 0.38833191990852356, 1.9026554822921753, -0.9591485261917114, -0.4441213607788086, 0.7197937369346619, -0.06793282181024551, -0.3998681306838989, 2.7608635425567627, 1.6036618947982788, -0.627919614315033, -0.6189988851547241, -1.501441240310669, 2.1782054901123047, 1.003589391708374, 0.2982137203216553, 0.6848016977310181, -0.8016815185546875, -1.3779253959655762, 1.8771507740020752, -0.20798921585083008, 2.3970870971679688, -1.216631531715393, -0.021933458745479584, -1.594665765762329, -0.3642275035381317, -1.5939399003982544, 1.4935959577560425, 1.065260410308838, 0.6395830512046814, -1.650749683380127, 0.16788391768932343, -0.21768158674240112, -0.06112686172127724], [1.3311196565628052, 0.9115157723426819, 1.5066553354263306, -1.5336663722991943, -0.26091721653938293, 1.4891148805618286, -0.08260966837406158, 0.55682373046875, -0.11768651753664017, 1.747728943824768, 0.9564008712768555, 0.3061639070510864, -1.9270683526992798, 0.4498840272426605, 0.8924022912979126, 0.2652739882469177, -0.7686991691589355, -1.1582841873168945, 0.7092544436454773, -0.36156243085861206, -0.05650641769170761, 0.8074440360069275, -0.3011893630027771, 1.3221126794815063, -0.33442243933677673, 0.10131001472473145, -0.9894979000091553, 0.2207462638616562, 1.140039324760437, -0.13893499970436096, -1.0049827098846436, -0.7141567468643188, -0.24245017766952515, -2.1341612339019775, 0.6699923276901245, 0.15162478387355804, -1.545658826828003, 1.4849989414215088, -0.4625494182109833, 1.4339747428894043, 0.827483594417572, -1.8947340250015259, 1.5565989017486572, -1.3451913595199585, 0.48356637358665466, 0.11863773316144943, -0.45940497517585754, 1.7554268836975098, 1.0003819465637207, 0.554490327835083, 0.14868775010108948, -1.0191736221313477, -0.22050565481185913, 0.5858874320983887, -0.44073405861854553, 0.14385710656642914, -0.32320311665534973, -0.5195935368537903, 1.2382901906967163, 0.17705640196800232, -1.7811754941940308, 0.5694164633750916, -0.5175754427909851, 0.27963340282440186], [0.14458328485488892, -0.1718892902135849, 0.35023269057273865, 1.5263237953186035, -1.2117244005203247, -2.8158810138702393, -1.9835014343261719, -0.06221886724233627, 1.7119840383529663, -0.65800940990448, -0.4784321188926697, -2.4961066246032715, 0.5304191708564758, 0.9810379147529602, 0.9983857870101929, -2.476679801940918, -0.255559504032135, -0.14158077538013458, -0.9088705778121948, 0.05158183351159096, -0.4116058945655823, -0.13930392265319824, -0.9976105690002441, -1.6112653017044067, -1.3336626291275024, 0.2733394503593445, -1.415228009223938, -1.779677391052246, 0.7371198534965515, 0.1463243067264557, 0.6159840226173401, -0.5166544318199158, -0.9904898405075073, -0.6749465465545654, -2.7157397270202637, -0.6876828074455261, -1.8696402311325073, -0.8689895868301392, 0.1554798036813736, 0.14734762907028198, 0.16347160935401917, -1.076587200164795, 1.355875015258789, 0.022782107815146446, -0.7449932098388672, -0.9083563685417175, 0.7344509363174438, -0.8190122246742249, -1.1286993026733398, 0.005376604851335287, -0.7687287926673889, 1.5685436725616455, 0.5723563432693481, -1.6616700887680054, 0.6124668717384338, -2.7834975719451904, -0.6587724685668945, 0.9883019924163818, -2.2151527404785156, 1.2246382236480713, 0.5527962446212769, 0.07002678513526917, -0.9978464841842651, 0.722683310508728], [-1.2281644344329834, 0.0034736287780106068, -2.0729258060455322, 0.007022043690085411, 1.576528787612915, 0.15715272724628448, 0.612230658531189, -1.3650418519973755, 0.12309318780899048, -0.9904654026031494, -0.40624353289604187, 0.6886159181594849, -0.7853609919548035, 1.1088769435882568, 0.914139986038208, 1.9073072671890259, -0.0792628824710846, 1.1829034090042114, -2.535902738571167, 0.39231905341148376, -1.9706604480743408, -1.0102006196975708, -1.1275359392166138, 0.5250394940376282, 0.4383881390094757, -0.48431333899497986, -0.06191825121641159, -0.7921218276023865, -1.31728196144104, 0.5552007555961609, -0.08057548105716705, 0.9214001893997192, 0.41499146819114685, -1.5371092557907104, 1.6107556819915771, 0.737657904624939, -0.5596214532852173, -0.33014658093452454, -0.6034364104270935, 0.2792339324951172, 0.15205004811286926, -0.2541770040988922, 0.7272992134094238, -0.30803635716438293, -0.4665572941303253, 1.8128353357315063, -0.9356997013092041, 0.7425975203514099, 0.37604835629463196, 0.9412291646003723, -1.0821418762207031, 0.2860892713069916, 2.6906328201293945, -0.2812032699584961, -0.6347672343254089, 1.1394050121307373, -0.39303886890411377, -1.201699137687683, -0.5304664969444275, -0.8471624851226807, -0.1034758985042572, -1.070113182067871, -0.2857564389705658, 1.0421509742736816], [0.2616196274757385, -0.7047603130340576, -0.35276955366134644, -1.2469621896743774, -0.6801612973213196, -0.2851223349571228, -0.3352433443069458, 1.0824846029281616, 0.33774715662002563, -0.13824446499347687, -2.5203094482421875, 0.5473991632461548, 0.771403968334198, -0.6005917191505432, 0.17165090143680573, -0.45842263102531433, 0.42671656608581543, 1.092839002609253, -0.5371608734130859, -1.0697481632232666, 0.4106603264808655, 0.5155162215232849, -1.6725431680679321, -0.06127281114459038, 0.6792059540748596, 0.9050077795982361, 1.2667717933654785, -1.088559865951538, 0.23084241151809692, -0.5121811032295227, -0.018166434019804, -0.49568817019462585, 0.8335148096084595, 0.333359956741333, -0.07152935117483139, -0.7912254333496094, -0.6673294901847839, -1.0379539728164673, 0.4936542212963104, 0.26100966334342957, 0.03483979403972626, -2.610578775405884, -0.9541787505149841, 0.7321429252624512, 1.3017363548278809, -1.3385109901428223, 0.07703178375959396, -0.08437180519104004, -0.3848218619823456, 1.1924093961715698, 0.9888678789138794, 0.5845202803611755, 1.3443951606750488, 0.3953307271003723, 1.4768366813659668, -0.01549439039081335, -0.7310823798179626, 0.20943567156791687, 1.9354588985443115, 1.3498998880386353, -0.9468262791633606, 1.05191969871521, -0.47282496094703674, -1.5781904458999634], [0.34841206669807434, 0.8035861253738403, -1.1791229248046875, 0.28345611691474915, -0.4883222281932831, -2.3105618953704834, 0.11159218847751617, -0.9287095069885254, 0.020275546237826347, -0.5628222227096558, -0.3983486592769623, 0.1031683012843132, 0.6103786826133728, 0.17945680022239685, -1.2893255949020386, -1.7013914585113525, 0.4410054087638855, -0.7455326318740845, 0.29801782965660095, 0.3259178698062897, -0.9327795505523682, -0.033263251185417175, 0.8994454145431519, 1.394060730934143, 0.817497730255127, 0.2689022123813629, 1.55963933467865, -1.8234905004501343, 0.053958021104335785, 0.4732506573200226, 1.7003358602523804, -1.0529731512069702, -0.5973424911499023, -1.4384328126907349, -1.2884161472320557, -0.01329986285418272, 0.15552091598510742, -0.3289167284965515, 0.06511389464139938, 0.6059143543243408, 0.8327681422233582, 0.4340405762195587, 1.4830758571624756, -0.5474783182144165, 0.39186036586761475, -0.35658156871795654, 1.2944693565368652, -0.6116084456443787, -0.6702373623847961, 0.4882955551147461, 0.3478530943393707, -1.2158013582229614, 0.21615926921367645, 0.7972524762153625, -0.9631754159927368, -2.2030773162841797, -0.9453762173652649, -0.6568838953971863, -0.9540506601333618, -0.10047993808984756, 1.5624310970306396, 0.794293224811554, 0.32586804032325745, 0.5724502801895142], [-0.39739474654197693, 1.4564231634140015, 0.0320449098944664, -2.0770833492279053, 0.3278075158596039, -0.45899391174316406, -0.3889346420764923, 1.8663064241409302, -0.8815078139305115, 0.1193419098854065, -1.8423185348510742, -0.4022551476955414, 0.41036805510520935, -0.4794183671474457, -0.22863686084747314, -0.7531827092170715, -1.8746134042739868, -0.3257874846458435, -0.018939929082989693, -0.8806490898132324, -1.8867920637130737, -0.33887091279029846, 0.025376036763191223, -0.06198359653353691, 0.05622730404138565, 0.06473363190889359, 1.7397725582122803, -2.0308077335357666, 0.013281266205012798, -0.6259697079658508, 1.3309193849563599, -1.3848302364349365, 0.2564579248428345, 0.968440592288971, 0.2830292880535126, 0.49041473865509033, -1.4014520645141602, 0.14218343794345856, 0.22174172103405, -0.9754757285118103, 0.3197857141494751, -1.1688737869262695, 0.5311002135276794, -0.161349818110466, -0.9128945469856262, -0.7529653310775757, 1.0722057819366455, 0.4997849762439728, 1.7319543361663818, 0.8983579277992249, 1.2158347368240356, -1.1506178379058838, -0.25715047121047974, 0.00951217021793127, 0.9936071038246155, -2.9088432788848877, 0.5261889100074768, -0.2514069974422455, -0.9131280779838562, -0.46506237983703613, -1.9898713827133179, -1.9359406232833862, -0.7403922080993652, -0.7683884501457214], [-0.1400260180234909, -0.30951201915740967, 0.378937691450119, 1.730522632598877, 0.05814095214009285, -1.5003094673156738, -0.7940307259559631, -1.7796392440795898, 0.6663932800292969, -1.1161963939666748, -2.1675329208374023, 0.9531634449958801, -0.4073755443096161, 0.5332685112953186, -1.3045451641082764, 1.368928074836731, -0.8903627991676331, -2.1037757396698, -0.7766767144203186, 0.08932038396596909, 0.23128753900527954, 1.4154661893844604, -0.6862762570381165, -0.8088138699531555, -0.3516280949115753, 1.9935200214385986, 0.15075840055942535, 0.6475164890289307, -0.3961191475391388, 0.38590100407600403, 1.3431954383850098, -1.1718451976776123, -1.5895870923995972, -1.1309016942977905, -1.3198028802871704, -1.469794511795044, -0.17326350510120392, 0.9336401224136353, 0.40428781509399414, -0.5245762467384338, 0.7341498136520386, -0.9402042627334595, -2.5111894607543945, -1.4357411861419678, 0.9686512351036072, -1.886920690536499, 0.9910857677459717, -2.310821771621704, 0.7171205282211304, 0.545021116733551, -0.1022520437836647, -0.2320508062839508, 0.25763174891471863, -0.3003499507904053, -1.3961330652236938, 1.3320552110671997, -0.9723324179649353, 1.2626687288284302, -1.6338557004928589, 0.05918693169951439, 0.7032530903816223, -0.5530577301979065, -1.4126684665679932, 0.12153078615665436], [-1.5493603944778442, 0.2080049365758896, -1.3324187994003296, -0.3491976261138916, -0.4283911883831024, 0.3324763774871826, 0.2654554843902588, 0.13654430210590363, -0.694442868232727, 0.22771242260932922, -0.41234615445137024, -0.7733526825904846, -0.015062312595546246, 0.9743863344192505, -1.9639111757278442, -0.7175238132476807, 2.0694754123687744, -0.1958964765071869, -1.8179036378860474, -0.4705010950565338, 0.4031822383403778, 1.151245355606079, -0.13116832077503204, 0.5849018692970276, -1.6000628471374512, 1.1416453123092651, -1.421384572982788, -0.24653899669647217, -1.8749933242797852, -0.20759309828281403, -0.384573757648468, -1.5163241624832153, -1.6476119756698608, 0.9321223497390747, 0.08803775906562805, -2.0260350704193115, -1.5490397214889526, 0.03671093285083771, 0.15389518439769745, -0.08456077426671982, -0.14050599932670593, 0.9875814318656921, -0.001732606440782547, 0.6977920532226562, -1.01316237449646, -0.33074241876602173, -0.4903481900691986, -0.1800697147846222, -0.38029369711875916, 0.716882050037384, -0.05573492497205734, -1.2396289110183716, 0.5146986246109009, 1.2923072576522827, -0.8224195837974548, 2.043086051940918, 0.40365472435951233, 0.25656551122665405, -0.5839598178863525, 1.1603904962539673, 0.13931205868721008, -1.011565089225769, 1.5383334159851074, -0.8019360899925232], [1.0090594291687012, 0.7742464542388916, 0.2061159312725067, -1.142022728919983, 1.0119363069534302, -0.4416353702545166, 0.024147113785147667, 0.14294511079788208, 1.8895999193191528, 2.0684397220611572, 0.5545940399169922, 0.8014909029006958, 0.2714902460575104, 1.8024139404296875, 0.6655887961387634, 1.2009530067443848, 2.8925957679748535, 1.0662871599197388, 0.38813450932502747, -0.6215149164199829, -1.6285531520843506, 1.4850139617919922, -0.8887937664985657, 0.083026222884655, -1.1285024881362915, -1.215946078300476, -0.030017441138625145, 1.469240427017212, 0.2703068256378174, -0.574852705001831, -1.8876301050186157, -2.603482723236084, -0.7887791991233826, -1.947876214981079, 1.9041179418563843, -0.26907452940940857, -0.42228779196739197, -0.09408438950777054, 1.3197318315505981, 0.3946191668510437, -1.9256882667541504, 1.758396029472351, -2.500776529312134, -0.6869086623191833, 1.735129714012146, 1.7306718826293945, 0.8226414918899536, -0.2611607313156128, -0.4399394989013672, -0.3817654252052307, -0.9025191068649292, -0.20236368477344513, 0.007134349551051855, -1.7932878732681274, -0.23395437002182007, -0.1783570498228073, 0.025722689926624298, -0.10288234055042267, 0.8303554058074951, 1.6837995052337646, 0.9590053558349609, 1.1164592504501343, -0.36191341280937195, -1.5916812419891357], [1.519607424736023, -0.5986608862876892, -0.13313154876232147, -0.3730437457561493, -0.0372428297996521, 0.4362023174762726, 1.3884855508804321, 0.8557490706443787, 0.4939228892326355, -0.32891151309013367, -0.6371113657951355, 0.317082941532135, -0.9605541229248047, -1.2338802814483643, -0.036904338747262955, 0.16148455440998077, -0.4203106462955475, -0.05084043741226196, -0.9739345908164978, -0.1302662044763565, -1.3745089769363403, -0.5196132063865662, -1.5444012880325317, -1.2655977010726929, -0.7182182669639587, 0.9172580242156982, -1.0597690343856812, -0.23744870722293854, -0.4216385781764984, -0.565187931060791, -1.2221895456314087, 0.2371559888124466, -0.4716435372829437, 0.12491068989038467, -0.8978980183601379, 0.3655901551246643, -0.5074553489685059, 0.4110087454319, 0.498660147190094, -0.9018749594688416, 0.33055630326271057, -1.099219799041748, 0.9311176538467407, -1.293312907218933, -1.101181983947754, -1.666981816291809, 1.3305588960647583, 1.1518620252609253, 1.623901605606079, 0.2532198429107666, 0.3610747158527374, -1.0263036489486694, 1.3080825805664062, 1.2703312635421753, -0.5515490770339966, -1.1207219362258911, -0.31249070167541504, -1.4917856454849243, 0.37131381034851074, -2.2114551067352295, 1.0018861293792725, -0.3335430324077606, 0.34792590141296387, 0.12656448781490326], [0.0939464196562767, 1.1348261833190918, -1.3951339721679688, 1.0893902778625488, -0.7583175301551819, -0.5933839082717896, 1.414480209350586, -0.46033626794815063, -0.04849648475646973, -1.892930507659912, -1.334256887435913, -0.37756484746932983, -0.6605538129806519, 0.4116387367248535, 1.4113465547561646, 0.16953270137310028, -0.45782795548439026, 0.0992632508277893, 1.588722586631775, -2.089128255844116, -0.19800381362438202, 0.2858445942401886, -0.1342024803161621, 1.3660370111465454, 0.803634762763977, 0.069198839366436, 0.5507972836494446, 0.5940165519714355, -1.1600667238235474, -1.2981536388397217, -0.5930973887443542, 0.5075693130493164, 0.46418502926826477, -0.6280273795127869, -0.3252411484718323, 0.6912265419960022, -0.2870030105113983, -0.5625783801078796, -0.6150566339492798, -0.48493990302085876, 0.8896198868751526, 1.1735676527023315, 0.4691765010356903, -0.19857487082481384, -1.4743965864181519, -0.22406697273254395, -0.9891878366470337, 0.02638867497444153, 0.3944041132926941, -0.8601744771003723, -0.5225238800048828, -1.6015056371688843, 2.7194037437438965, 1.1656068563461304, -1.954224705696106, 2.6903507709503174, -0.4643622636795044, -1.0295096635818481, 0.6464858651161194, 1.1018867492675781, -1.2280654907226562, 1.5184537172317505, -1.0294055938720703, -0.439035564661026], [-0.9278690218925476, 0.6325203776359558, -0.15285973250865936, -0.7065169811248779, 0.40911442041397095, -0.6578607559204102, 0.37435033917427063, 1.2685532569885254, 0.637272298336029, 0.8150296211242676, 0.6762864589691162, 0.18563206493854523, -0.4202645421028137, 2.122525453567505, 0.6854562759399414, -0.7927344441413879, -2.719878911972046, -2.2225406169891357, -0.11455778777599335, -1.0884257555007935, 0.277323454618454, -0.9308056235313416, -1.1511412858963013, 0.39180293679237366, -0.48821884393692017, 0.675057053565979, 0.01956084743142128, -0.8213362693786621, 1.0426594018936157, 1.255354881286621, -0.09511921554803848, 1.4049311876296997, -0.18820136785507202, -0.2787224054336548, -0.30141234397888184, 0.5375804901123047, -0.23185725510120392, -2.92667555809021, -0.352046936750412, -0.20886439085006714, 0.5869615077972412, -2.02630615234375, -0.023074980825185776, 1.5328450202941895, 0.020454538986086845, -0.9084293246269226, 0.2302693873643875, 0.3004365861415863, -1.050002098083496, -0.10144461691379547, -0.12290695309638977, -0.33245256543159485, 0.3028707206249237, 0.3699995279312134, -1.9635541439056396, 0.9647099375724792, 0.18873514235019684, 0.5347286462783813, 0.7673609256744385, -1.4558906555175781, -0.9405739903450012, -1.0576528310775757, 0.043381467461586, 0.9606733322143555], [0.9247934818267822, -0.13824462890625, 0.37066566944122314, -0.26054731011390686, -0.5279484987258911, 0.9640917778015137, 0.18842755258083344, 1.1517256498336792, -0.8838152289390564, 1.161420464515686, 0.48701193928718567, -1.0035609006881714, -1.7026467323303223, 1.127829909324646, -1.447892427444458, 0.4199126958847046, -1.9604166746139526, 0.25175416469573975, 0.41572126746177673, -0.8871261477470398, -0.5717029571533203, -0.91194087266922, 0.2814849019050598, -0.28246456384658813, 0.5359000563621521, -0.2692476511001587, 1.090950608253479, 0.9723126292228699, -0.8329014182090759, -0.6623519062995911, -0.3075716197490692, -0.4211556315422058, 0.7383592128753662, -0.5669972896575928, 1.6189442873001099, 0.5407124757766724, 0.17731419205665588, -0.4435095191001892, 0.9430931210517883, 1.4075419902801514, 0.4110024869441986, 0.5510386228561401, -0.604790985584259, -0.6305540204048157, 0.8478214144706726, 0.09879406541585922, -1.107073187828064, 0.37550947070121765, -0.024052871391177177, -1.9445124864578247, 1.0647342205047607, 0.8823406100273132, 0.30946630239486694, 0.5432239770889282, -0.09666641056537628, 0.9271582961082458, 0.1721879094839096, -0.8587985634803772, -1.0410995483398438, 1.3018730878829956, -1.7482720613479614, -0.7448973655700684, -0.4177663028240204, -1.7694445848464966], [1.7774014472961426, -0.8994919657707214, -0.05640313774347305, 1.124954342842102, -0.93455970287323, -1.8218259811401367, 0.2304535210132599, -0.37988507747650146, -0.3745552897453308, 0.4639861285686493, -0.02175208553671837, -0.7300275564193726, -0.4650552272796631, 0.8287339210510254, 0.20809979736804962, 0.7050275802612305, -0.629667341709137, 0.5789791345596313, 0.014217608608305454, 0.048475541174411774, -0.9720389246940613, -2.193341016769409, 2.5258114337921143, -0.815386176109314, -0.17854757606983185, 0.41120901703834534, 0.707099974155426, -0.4033251106739044, 0.43299490213394165, -1.3129503726959229, -0.9703352451324463, 0.9349057674407959, 0.9862631559371948, -0.405940979719162, -0.24161939322948456, -0.3059265911579132, 0.0008893460617400706, -0.06265193969011307, -0.23250775039196014, 0.6062226891517639, 2.2152247428894043, 1.7630640268325806, -0.5401616096496582, 1.3587003946304321, 1.3014488220214844, 0.5843588709831238, -1.0862703323364258, -0.2740239202976227, -1.5846270322799683, -0.7685571908950806, -0.20392337441444397, -0.37163856625556946, -0.1805305927991867, -0.7273135781288147, 1.4900224208831787, -0.14624451100826263, 1.3637412786483765, 0.897092342376709, -1.0414338111877441, -0.4506518840789795, -0.054435957223176956, -0.9501073956489563, 0.6196670532226562, -2.1528215408325195], [-1.0292658805847168, -0.2781311869621277, -1.159206748008728, -1.5466787815093994, 0.6570140719413757, -0.3402687609195709, -0.0950012281537056, 0.2442711591720581, 0.5929082632064819, 1.375432014465332, -0.08649114519357681, 0.573701024055481, 0.18862660229206085, -1.3100860118865967, -1.5270103216171265, -0.06634503602981567, 1.5280377864837646, -0.6801896691322327, -1.3121610879898071, 0.4762842357158661, -0.14778929948806763, -0.6170552968978882, -0.2604726254940033, 1.1043901443481445, -0.07717610895633698, -1.7154462337493896, -0.42607781291007996, -0.6985852122306824, -0.22959640622138977, 0.6340469717979431, 1.1563401222229004, 0.8603131771087646, -1.2059754133224487, 0.3252760171890259, 1.6832488775253296, -0.5858300924301147, -0.33543455600738525, -1.0197749137878418, -1.1682685613632202, -0.4316370189189911, -2.3003578186035156, -0.5797696709632874, -0.19239607453346252, -0.04642689600586891, -0.7964257597923279, 0.5823060870170593, 0.13237830996513367, 0.9593547582626343, 1.099109172821045, -1.3031827211380005, 0.5551227331161499, -0.28035515546798706, -1.4911463260650635, 0.08815654367208481, 1.1388717889785767, 0.38607507944107056, 0.7085149884223938, -0.6833134293556213, -0.2606544494628906, -0.8410813212394714, -0.4258284568786621, -0.5965058207511902, -1.2496408224105835, 0.11798935383558273], [-0.811683177947998, -1.237252950668335, 1.7757337093353271, 1.5980572700500488, 0.035572946071624756, 1.95863938331604, 0.6392178535461426, -0.7006980776786804, -1.583266258239746, 1.12492835521698, 0.5336670875549316, 0.43097615242004395, -1.615917682647705, 0.6365473866462708, -1.8074686527252197, -1.1076010465621948, 0.6225149631500244, 0.8045837879180908, -0.31555718183517456, 0.8818077445030212, -2.768071413040161, 0.7568577527999878, 0.46468624472618103, 1.6497504711151123, 1.1752288341522217, 0.3650394380092621, -0.5482591390609741, -2.569061040878296, 1.227080225944519, -1.223763346672058, 1.9938219785690308, 1.68217933177948, -0.8467908501625061, 0.8209500312805176, 0.32305026054382324, -0.023057589307427406, -1.4298392534255981, 1.461267352104187, -1.1481740474700928, 0.8752509355545044, -0.4721490740776062, -0.32673755288124084, 2.376695394515991, 0.07276005297899246, -0.26647424697875977, -0.6534379124641418, -0.34824785590171814, 2.1774189472198486, -2.2881875038146973, 1.602724552154541, 0.3010426163673401, 0.5377821922302246, -0.10760580003261566, -1.0730394124984741, 1.1158485412597656, -1.13117516040802, -0.40068405866622925, 0.49734458327293396, -0.1763773262500763, -0.19283805787563324, 0.1662374883890152, 1.7043936252593994, 0.5559305548667908, 1.589457392692566], [-2.0291011333465576, 0.7533127069473267, -1.5033962726593018, 0.44704383611679077, 1.0064697265625, 0.08976873010396957, 1.9231417179107666, -0.9984071850776672, 0.21837209165096283, 0.44380679726600647, -0.4463900625705719, 0.9816523194313049, 0.15463002026081085, 0.8723017573356628, 0.1289897859096527, -1.7208853960037231, 2.23844575881958, -0.26658135652542114, 0.4429182708263397, 0.996910810470581, -1.0990910530090332, 0.01020316407084465, 0.2754749059677124, 0.5462556481361389, -0.07719913125038147, 0.6016778945922852, -0.630137026309967, -1.4376243352890015, -0.44259366393089294, 0.2204650640487671, 0.6524670720100403, -0.20243658125400543, -0.40496426820755005, -1.6631516218185425, 0.24475212395191193, -0.9294300079345703, -0.0858507975935936, 1.7698664665222168, 0.3399586081504822, -0.88736492395401, 1.3640321493148804, -1.1572726964950562, -0.1408964842557907, 1.152263879776001, 2.136859178543091, 0.71865314245224, 0.9619132876396179, -0.8676418662071228, 0.497226744890213, 0.048846837133169174, 0.7398731708526611, -0.20743803679943085, -1.1943787336349487, -1.4051119089126587, 1.2933052778244019, 0.3801751136779785, -0.44843581318855286, 0.255587637424469, -1.1628412008285522, 0.0866350382566452, 0.06901666522026062, 0.07042383402585983, 2.881877899169922, -0.6920127272605896], [-0.6363140940666199, -0.0011153476079925895, -0.26562294363975525, -0.7222275137901306, -2.4824862480163574, -0.6688608527183533, -2.0932023525238037, 1.2532806396484375, -0.5332469940185547, 1.494079828262329, 2.7377588748931885, -0.4378657639026642, 0.9680408835411072, -1.2513985633850098, -0.33383744955062866, -0.5005141496658325, 1.0622901916503906, -0.33437487483024597, -1.3348755836486816, 0.5395192503929138, -0.6740989089012146, 0.3469432294368744, 0.49124205112457275, -0.7011895775794983, 0.6877465844154358, -1.0687671899795532, -2.5194344520568848, -0.2949010729789734, 1.971794843673706, 0.2743450403213501, 0.2628147304058075, 0.9683594703674316, -1.0437198877334595, -1.088695764541626, 0.4554319381713867, 1.008865237236023, -1.0244766473770142, 0.6480019688606262, -0.09446722269058228, 0.6233454346656799, 0.7282776832580566, -0.09751105308532715, -1.0655113458633423, 0.17119775712490082, 0.6368088126182556, 0.07243221253156662, -2.778489828109741, -0.16251610219478607, -1.357467532157898, -0.6571880578994751, 0.3410854935646057, 1.1859357357025146, -0.6247391104698181, -0.9090546369552612, 0.4222264885902405, 0.49238017201423645, -0.2049722671508789, -0.4092814326286316, -0.09746529906988144, -1.257561445236206, 0.8249294757843018, 0.2572392225265503, -0.2478748857975006, 2.6868064403533936], [0.29797378182411194, -0.2447078675031662, 0.952061116695404, 1.651928424835205, 0.307808518409729, -0.2736501693725586, -0.7333569526672363, -0.2882518172264099, 0.006440720986574888, 1.8384125232696533, -0.59774249792099, 0.3756878674030304, 0.39557260274887085, 1.0765275955200195, 0.645679771900177, -0.3659900426864624, 0.5610449314117432, -0.30359750986099243, 0.818270742893219, -0.8951530456542969, -0.924808919429779, -0.08319146931171417, 0.6063848733901978, -0.5832154154777527, 1.0826665163040161, -0.3579840064048767, -0.4539273977279663, 1.4539899826049805, 1.404928207397461, -0.5108765959739685, -1.1136029958724976, -0.9366589188575745, 0.029970744624733925, 0.45262590050697327, 0.8925375938415527, 0.44132471084594727, -0.4512478709220886, -0.253182053565979, 0.40623897314071655, -1.1981459856033325, 0.6388282775878906, -2.334568977355957, -1.3464088439941406, 0.17555178701877594, -1.2477153539657593, -2.3287699222564697, -0.6696677803993225, 0.9959072470664978, -0.5795022249221802, 0.1967138648033142, -0.18848921358585358, 0.4215489625930786, 1.718259334564209, -0.17595085501670837, -1.485244631767273, -0.8171780109405518, 0.6229219436645508, 0.10825270414352417, 0.6638902425765991, 2.5101757049560547, -0.9256070256233215, 0.9783512949943542, -1.8718972206115723, 0.811535656452179], [-0.663216769695282, 0.08248583972454071, -0.20543266832828522, 1.1345570087432861, 0.4582972228527069, -1.2286170721054077, 0.3607439398765564, -0.6785953044891357, 1.4687199592590332, -0.6188186407089233, -1.1424397230148315, 1.7553502321243286, -0.9499560594558716, -0.7011619210243225, 0.5712668895721436, -0.6357913613319397, -0.6085753440856934, -0.6235239505767822, 0.32951316237449646, -2.9338841438293457, 1.3034838438034058, -0.849185585975647, -0.877386748790741, -0.7889834046363831, -0.8210574984550476, -0.33613231778144836, -0.34873104095458984, -0.7861424088478088, -1.648358941078186, -0.0041479384526610374, -1.0598653554916382, -2.5599703788757324, 0.8731865286827087, 0.6419434547424316, -0.3413655161857605, -0.9874356389045715, -1.1592499017715454, -0.2615601420402527, -0.5705143213272095, -1.573347806930542, -0.09857496619224548, -0.8162881731987, 0.8402760624885559, 0.5243321657180786, 0.11639083921909332, -0.9627518057823181, -0.4099498391151428, 1.0822083950042725, -0.9126632213592529, -1.4614582061767578, 1.6813786029815674, 0.1271907538175583, 1.2866742610931396, -1.126354455947876, 0.31671133637428284, 1.8005013465881348, -1.596571922302246, -0.2693096697330475, -0.37310972809791565, -0.66087806224823, 0.6701927781105042, -0.46134379506111145, -0.49322474002838135, -0.015693217515945435], [0.6823620200157166, 2.2358899116516113, 0.7055665254592896, -0.23881462216377258, -1.4103437662124634, -0.8228821754455566, -0.04505420848727226, -0.8694155216217041, 0.6898375153541565, -0.4935418963432312, 0.7343339323997498, -0.32153868675231934, 0.7260039448738098, 0.16990543901920319, 0.479640930891037, -0.25565972924232483, 0.976631760597229, 0.8145696520805359, 0.7501225471496582, 0.9915466904640198, 0.7912600040435791, -0.46602028608322144, 1.3954869508743286, -0.3419797122478485, -0.8579617142677307, 1.8464826345443726, 0.970963716506958, 0.5288676619529724, 0.8315114974975586, -0.8055745363235474, 0.5032346248626709, -1.7407891750335693, 0.4059414565563202, 0.8480053544044495, -0.6601758599281311, 0.25876274704933167, -0.4660545289516449, -0.7360742092132568, -1.0886503458023071, 0.06927729398012161, -0.5623613595962524, -1.5660698413848877, -0.5767837166786194, 0.7967121601104736, 0.34261590242385864, -0.785620927810669, 0.5401728749275208, -1.1819279193878174, 1.0872242450714111, 1.119879961013794, 0.01724165864288807, 0.3864959478378296, 0.221351757645607, -0.2519192099571228, 1.4264658689498901, -0.9192457795143127, -0.6529731154441833, -0.3153093457221985, -0.03206745535135269, -0.4455934762954712, -0.6028285622596741, 0.16206307709217072, 1.19239342212677, -2.5753962993621826], [0.8721182942390442, 0.833182156085968, 1.3255220651626587, -0.7822009921073914, -0.62502521276474, 0.28990790247917175, -0.8048353791236877, -1.44011652469635, -0.015462060458958149, -0.3613283932209015, 0.20226149260997772, 0.5685592293739319, -1.5062735080718994, 0.724583625793457, -0.3254273533821106, -0.32866185903549194, 1.4634394645690918, 0.14998474717140198, 1.2189295291900635, 1.080520510673523, 0.2450578808784485, 0.8072514533996582, 0.9132686257362366, 0.9937981963157654, -1.240954041481018, -0.8081770539283752, -0.3917681872844696, -0.9408358335494995, 0.9744572639465332, 0.3920499086380005, 0.6381007432937622, -0.5547946691513062, -2.442610025405884, 1.804777979850769, 2.9146039485931396, -0.8850926756858826, -1.0832107067108154, -0.8212611079216003, 0.04389762505888939, 0.6394962072372437, 0.19935080409049988, -1.55777108669281, -0.16953305900096893, -0.8317974209785461, 0.31927424669265747, 1.0764786005020142, -0.788536548614502, 0.4263734221458435, 0.6024455428123474, -0.24619878828525543, -1.0133014917373657, -2.0628437995910645, 1.2629960775375366, -0.3299042880535126, 0.6944770216941833, 0.9668103456497192, -0.731740415096283, -0.4032280445098877, 0.5197668671607971, 0.47770169377326965, -1.5912572145462036, 1.3492865562438965, -1.0352035760879517, 0.1243835911154747], [-0.48453202843666077, -2.029384136199951, -0.5348795652389526, 0.11232997477054596, 1.0231847763061523, -1.1533399820327759, -2.0593976974487305, -0.39073646068573, 2.221407890319824, -0.6605632305145264, -0.2833554744720459, 0.4640938341617584, -0.7396852970123291, -0.10390156507492065, 0.024366816505789757, -0.007305356208235025, 0.6755040884017944, -0.33869585394859314, 0.4549708962440491, 0.5761181712150574, -0.8474093675613403, 1.3812237977981567, 0.5063451528549194, -0.4985331892967224, 1.1692512035369873, -0.9575952887535095, -1.7483052015304565, 0.10848955810070038, -1.6319702863693237, 0.06516602635383606, -0.9944213032722473, 0.23441527783870697, -0.49982336163520813, -0.642246425151825, 0.18615640699863434, -0.511003315448761, 0.1530994474887848, -1.4429579973220825, 0.3367353081703186, 0.6106969118118286, 1.1089125871658325, 0.6748952865600586, 0.7577295899391174, -0.12660884857177734, 0.009300070814788342, -0.06420882791280746, -0.4680008292198181, -1.9876364469528198, -0.016052164137363434, -2.440840244293213, 0.35912638902664185, -2.665778398513794, 2.4187586307525635, 0.31478846073150635, -0.7293755412101746, -0.5984535217285156, 0.47890686988830566, 0.9208233952522278, -0.743460476398468, -0.4291094243526459, 0.7716185450553894, -1.4878817796707153, -1.2160825729370117, -0.17593543231487274], [0.6913554668426514, 0.0011333636939525604, 0.9107790589332581, 1.3886759281158447, 0.8939067125320435, 2.2903025150299072, 1.342368483543396, 0.5062525272369385, 2.9731786251068115, 1.0852246284484863, 0.5808337926864624, -1.9803818464279175, -1.8920395374298096, -0.00587483448907733, -1.92621910572052, 0.3845987617969513, 0.12991422414779663, -1.5606883764266968, 0.08904042094945908, -1.1576426029205322, -0.08033975958824158, -0.004555453546345234, -0.21913890540599823, 0.8935297131538391, -1.1115280389785767, 0.0409667044878006, -0.7403644919395447, -0.9860279560089111, 0.6337296366691589, -1.1255244016647339, 0.004494714085012674, -0.21837492287158966, -1.294149398803711, -1.2267369031906128, 0.02269442193210125, -0.12061648815870285, -1.544700026512146, -0.8309610486030579, -1.5269397497177124, 0.6918451189994812, -0.6278622150421143, 0.19959145784378052, -1.2887086868286133, 1.215423345565796, -1.0679768323898315, -1.4276243448257446, 1.1260671615600586, 0.3902570605278015, 0.09799037873744965, 0.6260900497436523, 1.2383146286010742, 0.6089515089988708, -0.12406621873378754, 0.7693847417831421, 0.33996763825416565, 0.4939653277397156, 1.246683955192566, 0.33576127886772156, 0.17887116968631744, -0.1599574089050293, 1.2422116994857788, 0.22047393023967743, -0.3752194046974182, -0.3352595865726471], [1.2356878519058228, -0.5116279721260071, 0.5881426334381104, -1.3094760179519653, 0.8109911680221558, 1.972590684890747, 1.1272081136703491, 1.432037353515625, -0.16637037694454193, -0.10583455860614777, 0.9238857626914978, 0.010746447369456291, 0.8355887532234192, 0.19524401426315308, 0.4469431936740875, -0.7238180041313171, 0.630351185798645, 0.34790265560150146, -1.3474531173706055, 1.2742078304290771, 1.5951176881790161, 0.323055624961853, 1.3466272354125977, -0.49619364738464355, -0.07453517615795135, -1.1756149530410767, 1.2997151613235474, 0.27059757709503174, -0.29599621891975403, 1.3300822973251343, 2.9785287380218506, 0.30611079931259155, 0.08493832498788834, 0.7543480396270752, 0.4219423234462738, 0.5961307287216187, 0.8863844275474548, -1.1622847318649292, -0.016790760681033134, -0.3788232207298279, 1.3922832012176514, 1.3234875202178955, 0.07856162637472153, -1.3879914283752441, -0.2555355727672577, -1.1248903274536133, -0.19314348697662354, 0.4053223431110382, -0.6742268800735474, -0.6923720240592957, 0.22786490619182587, 1.287198543548584, 1.0238981246948242, 1.5731620788574219, 0.4158324897289276, -0.7373071312904358, 1.135249137878418, 2.092482089996338, -1.1997102499008179, -1.5658835172653198, -1.0709660053253174, 0.958372950553894, 0.7204482555389404, -0.00989673100411892], [-1.450293779373169, 1.98567533493042, 1.2025644779205322, -1.5459256172180176, 0.39438313245773315, 0.548976480960846, -0.326389878988266, 2.4089229106903076, 0.1661987602710724, 0.9360279440879822, -1.5987907648086548, -0.8966164588928223, 1.1342183351516724, -0.15249942243099213, -0.14068640768527985, -0.23548609018325806, -0.5395066738128662, -0.8956859111785889, 1.8010107278823853, 0.8763008117675781, -0.12016860395669937, 0.1698109656572342, 1.6421878337860107, -0.01661519519984722, 0.5850223898887634, -0.19994094967842102, 0.18303491175174713, 1.3058674335479736, -0.668789267539978, 0.41799670457839966, -0.7758944630622864, 1.2240309715270996, -0.8420310616493225, 1.6842809915542603, 0.7783055901527405, -0.10311286896467209, -0.1690029352903366, -0.25670138001441956, 2.5207955837249756, -0.4491100311279297, -0.012037915177643299, -0.1995219886302948, 0.3799608647823334, -1.8986947536468506, 2.0756683349609375, -0.8519688844680786, 1.4886207580566406, 0.42754241824150085, 2.776662826538086, -1.1348577737808228, 0.0010333812097087502, -0.4867076575756073, 0.05140466243028641, 1.681850790977478, 0.36806246638298035, -0.14497625827789307, 0.7758438587188721, 0.7323654890060425, -0.30257704854011536, -0.39990514516830444, -0.19739463925361633, -1.2293503284454346, -1.5779480934143066, -1.6028382778167725], [2.5067954063415527, 0.8188193440437317, -0.343467652797699, 1.7656611204147339, -1.1034696102142334, 1.1165494918823242, 0.35003483295440674, 0.6706831455230713, -0.23803280293941498, 0.7272611260414124, 0.38335785269737244, 1.3006958961486816, -0.5137254595756531, 0.5611725449562073, 1.8492923974990845, 1.2945728302001953, -1.4389030933380127, -0.21882763504981995, 0.3510158658027649, -0.9790267944335938, -1.1804635524749756, 0.48582616448402405, -0.43075308203697205, -0.35057103633880615, 0.06491247564554214, 0.5049639344215393, 1.3293293714523315, -0.711326539516449, -1.1545027494430542, -1.748699426651001, -1.1525572538375854, -0.9535326361656189, -0.15631863474845886, 1.1067790985107422, 0.5887609124183655, -0.5667181611061096, -0.26584529876708984, -0.2262454777956009, 0.2584899663925171, 0.8373843431472778, 0.4667258560657501, 0.35118958353996277, -0.13191954791545868, -0.9218001961708069, -0.45458558201789856, 1.177180290222168, 1.1968938112258911, 0.021217359229922295, 0.7527886629104614, -1.6960647106170654, 2.6132822036743164, -0.21291649341583252, -1.0864726305007935, -1.257095217704773, 0.37537267804145813, 1.5727490186691284, -0.2343517690896988, -0.34001561999320984, -1.5719952583312988, 1.1581380367279053, 2.4367523193359375, -1.1853013038635254, 1.0129245519638062, 2.501351833343506], [-1.039634108543396, -0.6261903643608093, -0.8060347437858582, 0.95562344789505, 1.1648356914520264, -0.7081519365310669, 1.9373408555984497, -0.1420583873987198, -1.7053439617156982, -1.7009400129318237, 0.1294977217912674, 1.2621047496795654, 0.003249012166634202, 0.21548883616924286, 1.0446240901947021, 0.587814450263977, -0.9699946641921997, -0.9359157085418701, -1.3116610050201416, 0.5719245672225952, 0.22377634048461914, 0.9626774191856384, -1.0360468626022339, 0.9387863874435425, -0.48471397161483765, 0.37646323442459106, 0.10961131751537323, -1.076727032661438, 0.5554583668708801, -0.40241339802742004, -0.10235551744699478, -1.3900583982467651, -0.5728961825370789, 1.241736650466919, -0.872194766998291, 0.7295054197311401, -1.1162585020065308, -0.49083611369132996, 0.9792510867118835, 2.065361499786377, 1.2850346565246582, -2.0401275157928467, -1.452065348625183, 1.4553025960922241, 1.568602442741394, -0.04653177037835121, -0.961576521396637, -1.9162571430206299, -1.2240824699401855, 0.03291535750031471, 0.23941196501255035, -0.9150975346565247, -0.10090259462594986, 2.0992469787597656, 0.939754068851471, -1.1903961896896362, -1.5882130861282349, -2.4165427684783936, 1.4744938611984253, -0.7222487926483154, -1.957377552986145, -0.15662653744220734, 0.23552918434143066, -1.0701258182525635], [0.6626284718513489, -0.9411734342575073, -0.6994566917419434, 0.5929569005966187, 0.12848317623138428, -1.739649772644043, 0.2897823750972748, 1.3711988925933838, -0.6403390169143677, -0.5879050493240356, -1.4269018173217773, -0.43690115213394165, 1.9337671995162964, 0.966935932636261, -0.5617562532424927, -1.0652142763137817, -0.5021926164627075, -0.3567564785480499, -0.9763166904449463, -1.993265151977539, 0.45244598388671875, 1.1671960353851318, 1.4386991262435913, -0.2895122766494751, 1.2305527925491333, 0.9841084480285645, -0.6040809154510498, 1.35944402217865, 0.09685312956571579, -0.3672383725643158, 1.5949079990386963, -0.06452810019254684, 0.19380852580070496, 2.1318066120147705, -1.4403935670852661, 0.2423921525478363, -1.6250710487365723, -0.36892786622047424, -0.3555094301700592, -1.0909910202026367, 0.337462842464447, -1.0886725187301636, 0.16739137470722198, -0.7867192625999451, 2.0955469608306885, -0.6305662989616394, 0.9918941855430603, -1.2281290292739868, -0.001912781735882163, -0.8405742049217224, -0.9694876074790955, 0.682964563369751, 0.7076911330223083, -1.6924209594726562, -0.6582970023155212, 0.7777471542358398, 0.858958899974823, 0.5165363550186157, 2.481548309326172, -0.8143175840377808, 0.92545086145401, -0.2821962237358093, -0.7979426383972168, -0.7075299620628357], [0.5630245804786682, -0.4503629803657532, 0.09677023440599442, -0.21629966795444489, -0.7315927147865295, -0.42094293236732483, 0.6017530560493469, -0.1358049213886261, 0.5246956944465637, 0.13004998862743378, 2.1975533962249756, 1.1218287944793701, 0.3618432283401489, -0.9727028012275696, 0.08533167839050293, 0.2959982454776764, 0.8161739706993103, -0.5561458468437195, 0.9714396595954895, -1.245867133140564, -1.3015955686569214, 0.7640771865844727, -0.2326093316078186, -0.6805035471916199, -0.989571750164032, 0.11047861725091934, -1.2979027032852173, -1.8510037660598755, 0.22764238715171814, 0.6969423294067383, -0.4657663106918335, 0.8389797806739807, 0.9157423377037048, -1.0840238332748413, 0.760951817035675, -0.032116517424583435, 1.971490740776062, 1.3307650089263916, -1.1518077850341797, -0.09726617485284805, -0.37693309783935547, 1.011408805847168, 1.2251005172729492, 0.2467740923166275, 1.6859817504882812, -0.5005557537078857, -1.4180222749710083, -1.194793701171875, -0.27597513794898987, -0.7101384401321411, 0.020904652774333954, 0.5927963852882385, -1.0749765634536743, 0.08893871307373047, 0.4582255780696869, 0.1640947312116623, 0.04802447557449341, -0.5585716366767883, -0.005828481633216143, 2.8570775985717773, -0.7113252878189087, -1.009644627571106, -1.9838842153549194, -0.5216600298881531], [-1.1294323205947876, -1.3046952486038208, -0.7403002381324768, 0.04020475223660469, -1.2544525861740112, -0.6868312954902649, 0.4681553244590759, -0.6401088237762451, -0.23892274498939514, 1.0152581930160522, -1.3035390377044678, 0.8731414675712585, 0.46417057514190674, -0.2335941642522812, -0.5065677165985107, -1.1204557418823242, -0.840097188949585, -1.3269821405410767, 1.5116716623306274, -0.9749930500984192, 0.573969841003418, 0.2985938489437103, 0.5582903027534485, -0.9749181866645813, 1.6679915189743042, 0.29209646582603455, 1.159848690032959, -0.33745792508125305, 1.017265796661377, 0.20801027119159698, 1.0228945016860962, -0.9994834661483765, -0.6446447372436523, -0.7201525568962097, 0.19423764944076538, -1.2126010656356812, 0.9349944591522217, -0.49196818470954895, -1.1907892227172852, 0.7663004994392395, -0.0704815685749054, -0.9799473881721497, 0.12475772947072983, 1.6704587936401367, -1.3860595226287842, 0.749285101890564, -0.8974336385726929, -1.6023290157318115, 0.2870151102542877, 0.7926393747329712, 0.10867060720920563, -1.0634690523147583, -0.3256829082965851, -0.9648904800415039, 0.21043740212917328, -1.9566627740859985, -0.09962420910596848, -1.054481029510498, 1.1322308778762817, -1.7686694860458374, -0.27718988060951233, -0.6424738764762878, -0.6014246344566345, -1.0518114566802979], [-1.2979052066802979, -0.833486795425415, 0.3821127414703369, -0.8256551623344421, -1.521296739578247, -0.4198341369628906, -1.3285324573516846, -0.6979373097419739, 0.9526200294494629, -1.111811876296997, 0.008336147293448448, 0.5228853225708008, 0.18266965448856354, -1.926200270652771, 0.927692174911499, -0.9217888712882996, -1.1102650165557861, -1.5108528137207031, 0.6395120024681091, -0.9633427858352661, 0.3059428930282593, 0.3599155843257904, -0.40729570388793945, -0.985802948474884, -0.11776319891214371, 0.4563741683959961, 2.2769412994384766, 2.7485098838806152, -1.7279328107833862, -0.10823113471269608, -1.1118353605270386, -0.0749044120311737, -0.239558607339859, -0.579007625579834, -0.428846538066864, 0.10120616108179092, 0.13071882724761963, 0.41258108615875244, 0.13632206618785858, 1.3741487264633179, -1.7563238143920898, -1.0235402584075928, -1.2260600328445435, -1.4379496574401855, 0.02963659167289734, -0.4037253260612488, 0.5371579527854919, -1.5482704639434814, -2.622821092605591, 0.15521685779094696, -0.9922094345092773, 0.020881323143839836, 0.5985985398292542, -1.671531319618225, -0.11912117898464203, 0.11344844847917557, 1.086475133895874, 0.7470679879188538, -0.1603195071220398, -0.39887183904647827, -1.9580185413360596, -0.6237688064575195, 0.6301101446151733, 2.085557222366333], [-0.19818353652954102, 0.0838981494307518, -0.27594321966171265, 0.9550703167915344, 1.5726715326309204, -0.28526532649993896, 1.50100839138031, 0.5339269042015076, -0.8580788969993591, -1.8430087566375732, -0.5874560475349426, -0.7504934072494507, 0.7968403100967407, 1.7567042112350464, -0.4317346215248108, 0.6140491962432861, 1.2153537273406982, -0.19109973311424255, 0.1266457438468933, 0.5465183854103088, 0.8028615117073059, -0.436922550201416, -0.8888179063796997, 0.19465936720371246, -0.19288264214992523, 1.373152732849121, 0.49907395243644714, -1.6865202188491821, -0.9411804676055908, -0.3861234486103058, 0.27160772681236267, 0.7465589642524719, -1.0798389911651611, 0.17134207487106323, -1.3428218364715576, 0.10203047096729279, 0.2514585852622986, -0.9938040375709534, 0.7986049652099609, 1.3570727109909058, -0.666246235370636, -1.3840365409851074, -0.825888454914093, -0.8010976910591125, -0.9530344009399414, -0.6150549650192261, -1.5622937679290771, 1.2725310325622559, -0.6695037484169006, -0.13069991767406464, 0.6674460172653198, 0.26790139079093933, -0.14444710314273834, 1.3658021688461304, 0.6272749900817871, 1.2469993829727173, 1.3694233894348145, -1.1174147129058838, -1.3568350076675415, 0.7559648156166077, -0.4895642399787903, -0.06018543243408203, 0.07597105205059052, 0.3738754391670227], [-1.4739314317703247, -0.06834795325994492, -0.5044532418251038, -0.7355424165725708, -0.012587280943989754, -0.7341719269752502, -2.1295695304870605, -0.5837580561637878, 0.3268650770187378, 1.3685837984085083, 0.20924419164657593, 1.3566925525665283, -0.2150476723909378, 0.4290519058704376, -0.45619848370552063, 0.10241065174341202, 0.32358089089393616, -0.219418004155159, 1.0177102088928223, -2.272667646408081, -0.645645260810852, 2.4005966186523438, 1.0326886177062988, 0.1747485101222992, -2.2922561168670654, -0.7750643491744995, -0.1852405071258545, -1.3594286441802979, 0.04187172278761864, 0.5672820210456848, -0.45995503664016724, 0.5580708980560303, 0.13101133704185486, -1.232962965965271, 0.5655749440193176, 1.000183343887329, -0.3969151973724365, 0.5804104804992676, 0.2678362727165222, -1.3933228254318237, 1.6560081243515015, 0.708800196647644, -1.8188520669937134, -1.5109144449234009, -0.3117232322692871, -0.4153117537498474, 0.1769217699766159, -0.08331426978111267, -0.7066524624824524, 0.4630633294582367, 0.02615833282470703, 1.1830374002456665, -0.501310408115387, -0.42334800958633423, 1.3406431674957275, 2.3904833793640137, 1.1769403219223022, -1.0000687837600708, -0.6883490085601807, 0.7660089135169983, -0.9928570985794067, 2.5518076419830322, 0.3403307795524597, 0.7354828119277954], [-1.020422101020813, 0.3994913101196289, 0.6150177717208862, 0.30971771478652954, -0.3966808021068573, -0.5284338593482971, -0.7685510516166687, 0.8769056797027588, 0.5603482723236084, 0.5262026190757751, -1.4759929180145264, 0.46519723534584045, -0.7746595144271851, 0.6735155582427979, -1.9259861707687378, 0.4381902813911438, 0.47165197134017944, 0.5006105899810791, 0.846652626991272, -0.29416701197624207, -0.3902321755886078, -0.5720826387405396, 0.38192325830459595, 1.0697388648986816, 0.6237437725067139, -1.2958850860595703, 0.05182652920484543, -0.7841864228248596, 0.5895086526870728, 0.33582645654678345, -0.4514527916908264, -0.2602629065513611, 0.3220631182193756, 0.8534293174743652, 0.4046546518802643, 0.09103429317474365, 0.7763127684593201, -0.4071792662143707, 1.560238003730774, 1.0424556732177734, -0.48526206612586975, 1.2142488956451416, 0.050126124173402786, -1.3335353136062622, 0.4667952358722687, 0.06934870034456253, -0.7267605662345886, -2.0228078365325928, -0.7259815335273743, -1.4080400466918945, 0.04640776664018631, -0.5526131391525269, -0.5635067224502563, 1.5348296165466309, -1.92787766456604, 0.339339941740036, -0.6842271685600281, -0.08411502838134766, -0.5506766438484192, 1.6707184314727783, 0.17528533935546875, 1.0557246208190918, -1.4534939527511597, 1.262563705444336], [-0.033746495842933655, 0.4655427038669586, -0.5722329616546631, -0.6699232459068298, 0.32937148213386536, 2.1888792514801025, 0.459151953458786, 0.9170050621032715, 0.5865932703018188, -0.44589483737945557, 0.09506477415561676, 0.3611924648284912, -0.7119459509849548, 0.696498453617096, 0.7849327325820923, -0.8267194032669067, 0.0671868696808815, -0.6498609781265259, 0.1356961876153946, -0.9731530547142029, 3.640012264251709, -0.9718806743621826, -1.6438815593719482, -0.42306065559387207, -0.8366768956184387, -0.1758902221918106, -0.40451934933662415, 1.8951988220214844, 0.3758387863636017, -0.6962218284606934, 1.6598565578460693, 0.5684396028518677, -1.2251304388046265, -0.00043769855983555317, -0.7905585765838623, 1.011283278465271, -0.9141935110092163, -1.1370129585266113, 1.0213559865951538, 0.619367778301239, 1.357602596282959, 0.11999625712633133, 0.11410234868526459, -0.6506746411323547, -0.3438766300678253, 0.5060558915138245, 1.1777864694595337, 0.11269880831241608, 2.33026123046875, 2.2940351963043213, -0.4367680251598358, -0.47293201088905334, 0.15164819359779358, 0.008784646168351173, 0.1883593499660492, 2.5049095153808594, 1.6987055540084839, 1.806947112083435, -1.2595118284225464, 1.5783025026321411, 0.7731438279151917, 0.9938013553619385, 0.40721747279167175, 0.7023167610168457], [-0.6414003968238831, -0.9016494154930115, 0.4699242413043976, 0.32823988795280457, -0.27150243520736694, 0.9501569867134094, -0.7369244694709778, -0.022442670539021492, -1.0718038082122803, 1.336013913154602, -0.23670440912246704, -1.2214807271957397, -0.5861626863479614, 0.2377042919397354, 0.45321303606033325, -2.2531943321228027, -0.3146078586578369, -1.0987906455993652, -0.027366045862436295, 1.2525722980499268, 0.34659093618392944, -1.7909730672836304, 0.37411484122276306, 0.7498550415039062, 1.4745686054229736, -0.303554505109787, 0.1778680980205536, -0.558364987373352, -0.9420273303985596, 0.7507380843162537, -0.5063947439193726, -0.8659204244613647, -0.7765677571296692, 0.5013445019721985, -0.2734138071537018, 0.6156032681465149, -0.3120427131652832, 0.8666667342185974, -0.34921547770500183, -0.38255780935287476, -0.37509623169898987, 1.1325862407684326, 1.1473608016967773, -0.27139946818351746, -1.5913835763931274, -0.1660856157541275, 0.804288923740387, -0.560847818851471, -0.011852873489260674, 1.3212037086486816, -0.04028184339404106, -0.14097383618354797, -1.5964632034301758, 0.677855372428894, -0.12303610146045685, -0.3411974608898163, 0.6170300245285034, 1.1929112672805786, -0.12273042649030685, -0.4469612240791321, -0.7364916205406189, 1.0952706336975098, 1.269182562828064, 0.40076887607574463], [-0.042218971997499466, 0.8517014384269714, -0.01931162178516388, 0.5596131086349487, 0.7577744722366333, -0.3909946382045746, 2.3742575645446777, -1.0433440208435059, 0.38142645359039307, -0.11245669424533844, 0.39290937781333923, 0.8352013826370239, -0.2413949966430664, 0.20525579154491425, -0.9709769487380981, 0.5377833843231201, -0.266915500164032, 2.3331148624420166, -1.0435621738433838, 1.3748321533203125, -0.6714897751808167, -0.1065477579832077, 1.6943320035934448, -0.5045088529586792, -0.28474491834640503, -0.22805267572402954, 0.8164456486701965, -0.4353334307670593, 0.4776114523410797, -0.059426527470350266, -0.047300178557634354, 0.6621688604354858, 0.8800053596496582, 1.423134207725525, 1.5635857582092285, 0.3802112340927124, -1.7417048215866089, 0.4695119857788086, 0.5635627508163452, 1.565309762954712, -0.17555661499500275, 0.43032029271125793, 0.147599995136261, 1.521541953086853, 0.38085508346557617, -0.9705770611763, -0.27444180846214294, -0.015348331071436405, -2.4237396717071533, 0.2886020243167877, 0.7394019961357117, -0.15472270548343658, 1.2179948091506958, -1.0356537103652954, -0.30739784240722656, -0.07259684056043625, 1.7956953048706055, 1.1834464073181152, -0.9077928066253662, 1.2242636680603027, 0.3597431778907776, 1.4840253591537476, 0.15909507870674133, 0.49886438250541687], [0.7072175145149231, 1.8679969310760498, -0.6029601693153381, 0.5483395457267761, -1.769675612449646, 1.1104722023010254, -0.953605055809021, 0.53722083568573, -0.7979600429534912, 0.7298780679702759, -2.2042126655578613, 0.031880103051662445, 0.07407497614622116, 0.5055357217788696, 0.029388757422566414, -0.1988663226366043, 0.06927920877933502, 0.6058764457702637, 0.20889399945735931, 0.6255717277526855, -1.5123907327651978, 0.2705830931663513, 0.8916949033737183, -1.5247353315353394, -0.6912854313850403, 0.43160519003868103, 1.003390908241272, 1.9975508451461792, -1.535668969154358, -0.26610201597213745, -1.2566591501235962, -0.6774688363075256, 0.17329780757427216, -0.4833143651485443, -1.207344889640808, 0.14201435446739197, 0.053257618099451065, -0.4264760911464691, 2.078061580657959, 1.2747859954833984, -0.6375083327293396, -0.14407116174697876, -0.23768241703510284, 0.05564955621957779, 1.491516351699829, -0.7140653133392334, -0.6507729291915894, -1.0371555089950562, -1.2532093524932861, -0.4165613055229187, 1.1892940998077393, -1.1110392808914185, -1.1744168996810913, 0.15046104788780212, 0.7116694450378418, -0.8165956139564514, -1.298904538154602, -0.46680983901023865, -0.40970996022224426, -0.1041158139705658, -1.2945979833602905, -1.7760744094848633, -0.719281792640686, 0.3045600652694702], [-1.3917101621627808, 0.6531190872192383, -0.1631031185388565, -1.3439909219741821, 0.01662745326757431, -0.37873223423957825, 0.06840132176876068, 1.25899076461792, -0.8795077204704285, 0.6649026274681091, -1.8650864362716675, -0.2796884775161743, 0.9949328899383545, 0.3878540098667145, -1.406410813331604, -0.8972309827804565, -1.6441924571990967, 0.3678865134716034, -0.46352311968803406, -0.5559123754501343, 0.3040945529937744, -1.8710447549819946, 1.549243450164795, 0.21294188499450684, 0.4665140211582184, -0.2767556309700012, 1.3933179378509521, 0.46034136414527893, 0.8706310391426086, -0.8963844180107117, -0.7878608107566833, -2.1259348392486572, -0.001810609013773501, -0.4214418828487396, -0.5834142565727234, -0.5388416647911072, 0.4581833779811859, 0.5823585987091064, 1.5745480060577393, -0.356101393699646, -1.086690902709961, 1.2614134550094604, -0.10284914076328278, -0.38986721634864807, 0.14702987670898438, 1.1559064388275146, -1.3038618564605713, -0.44475752115249634, 1.5607191324234009, 0.07676126807928085, 1.7440600395202637, 1.9253615140914917, 0.5725287795066833, 0.038149550557136536, 0.9732861518859863, 0.9911529421806335, 1.433172583580017, -0.8466342687606812, 1.5409138202667236, -1.0798492431640625, 1.2240113019943237, 0.0026033809408545494, 2.8197109699249268, -0.7623417973518372], [0.2565768361091614, 0.09099498391151428, -0.06103387475013733, -0.9343032240867615, -0.023789305239915848, 0.38533836603164673, -0.49895238876342773, -0.2773750126361847, 1.3547699451446533, -0.1891753226518631, 0.14142248034477234, -0.31338000297546387, -0.9953513741493225, -0.14303988218307495, 0.4657299518585205, 1.0458515882492065, -1.474560260772705, -1.5938791036605835, 0.9754965901374817, 0.5180050134658813, 1.3176329135894775, 0.09220436960458755, -0.24219977855682373, -0.8499265909194946, 1.020804524421692, -0.48424893617630005, 1.6833813190460205, -2.119236707687378, -1.1708709001541138, 0.6087137460708618, -0.667391300201416, 1.1839871406555176, -1.1957584619522095, 0.240819051861763, -0.9596003293991089, -0.8843024969100952, 0.4775983989238739, 1.1275551319122314, 1.3794927597045898, -1.7369449138641357, 0.5768827199935913, 1.2113827466964722, -0.5257394909858704, -0.27570876479148865, 1.303847312927246, 0.932670533657074, 1.913201928138733, 1.2750685214996338, 0.42520323395729065, 0.20900148153305054, -0.24545517563819885, -0.7759989500045776, -0.9897461533546448, -0.5292655825614929, -0.6320606470108032, -0.25936198234558105, 0.15989354252815247, 1.1946278810501099, -0.1434888392686844, -0.4107934534549713, 0.32158756256103516, -0.6322324275970459, 0.27114614844322205, 0.3906930685043335], [1.094664454460144, -0.15469008684158325, -0.47842276096343994, 0.5817312002182007, -1.2731187343597412, -0.23252588510513306, 0.29121342301368713, -1.1026341915130615, 0.2772250175476074, 0.9397506713867188, -0.445517361164093, 1.0229114294052124, -0.8064009547233582, -1.9935144186019897, -1.6343305110931396, 0.4322621524333954, -0.03587622940540314, -1.8032952547073364, 0.9597898125648499, -0.4945216774940491, -0.6104499697685242, -0.2283245474100113, 0.9496974945068359, 0.6805552244186401, 0.9830806851387024, 0.08245394378900528, 0.5606422424316406, 0.4322394132614136, -0.9374805688858032, 1.0154144763946533, 0.017329201102256775, -0.2529756426811218, -0.45792654156684875, -0.3173314034938812, -1.3350697755813599, 0.37723878026008606, -1.9151231050491333, 1.249668836593628, -0.46243372559547424, -0.4105257987976074, 0.7573995590209961, -0.3038083612918854, -1.09745454788208, 1.5601452589035034, -0.5058937668800354, 1.3728556632995605, 2.60099458694458, -0.7056497931480408, -0.23117676377296448, 0.8765802979469299, 0.9782852530479431, -1.1947332620620728, 1.8397860527038574, 1.6701278686523438, 0.13386468589305878, 1.9593719244003296, -1.646684169769287, -1.5082224607467651, 0.1742209792137146, 1.2385727167129517, -1.5193191766738892, 1.259618878364563, -0.44638168811798096, 0.005511384457349777], [0.18040668964385986, 0.2506435811519623, 0.5577377676963806, 1.7468570470809937, 1.388832688331604, -1.1598409414291382, 1.0077483654022217, 0.16862818598747253, -0.23513607680797577, -1.7568306922912598, -1.8225057125091553, -0.6631976962089539, 0.43252164125442505, -0.23620808124542236, 0.6115500926971436, -1.538277506828308, -0.5082193613052368, -1.2581896781921387, 0.7618682384490967, 0.3285408020019531, -1.3734462261199951, 2.143289089202881, -0.7119092345237732, -1.4527186155319214, -0.6761154532432556, 0.2571590542793274, -0.1849418580532074, 0.23388683795928955, -0.3352239429950714, 0.5429723262786865, -0.25454291701316833, 1.271978497505188, 0.6943516731262207, -0.03246907889842987, 0.6609381437301636, 0.8009848594665527, -2.040632486343384, -0.27687495946884155, -0.18087632954120636, -0.06526438891887665, -0.02261010929942131, -0.6949559450149536, -1.1660653352737427, 0.02014804072678089, 0.18246549367904663, -0.0033530069049447775, 0.3539040684700012, -1.1432427167892456, 1.8511008024215698, -0.03388284146785736, -0.4864218235015869, 0.17413337528705597, 1.1516308784484863, 0.893550455570221, 0.6088008880615234, -0.2764946222305298, -1.344161868095398, -1.2485716342926025, 0.37342599034309387, 1.462152123451233, 1.261383056640625, -0.25421538949012756, -1.4884555339813232, -2.1394033432006836], [-0.4435840845108032, -0.8904601335525513, -0.07898424565792084, 0.1924259215593338, -1.8416812419891357, 1.7507002353668213, 2.413165330886841, -0.45214828848838806, -0.2059430629014969, 0.4108000099658966, -0.38013342022895813, -0.3994632959365845, -0.295820027589798, 0.9500340223312378, 0.4895347058773041, 0.5790584683418274, -0.6918282508850098, -1.1458371877670288, 2.9135162830352783, 0.22670310735702515, -0.22853153944015503, -1.31248939037323, -1.0168416500091553, 0.6029366254806519, -0.01117404829710722, -2.0626533031463623, 0.4774799942970276, -0.4981360137462616, -0.6507518887519836, -0.5037652254104614, -0.4653906226158142, 0.18062736093997955, -0.3163856565952301, -0.8541253209114075, -0.7586408853530884, -0.12173270434141159, -0.27938106656074524, 0.9534706473350525, 0.06868155300617218, 1.1134401559829712, 1.2535200119018555, -1.36233651638031, -0.6657348275184631, 0.0685228481888771, -0.4953591227531433, -0.2983677387237549, 1.8187150955200195, 0.84871906042099, -1.2223018407821655, 0.24005450308322906, 1.7214031219482422, 0.36891427636146545, -0.7873938083648682, -1.3951209783554077, -1.7371317148208618, -1.901149034500122, 0.27925193309783936, 1.1508557796478271, 0.5206451416015625, -2.685316562652588, -1.4068200588226318, 2.0487797260284424, -0.31467369198799133, -0.6850497722625732], [0.3932907283306122, -0.838458240032196, -0.19479988515377045, 0.7715827226638794, -0.7976968884468079, -2.29378342628479, 0.26222383975982666, -0.18092766404151917, 0.3106643855571747, -1.0570805072784424, 2.17270565032959, -0.42055585980415344, -0.4090828001499176, 0.5144853591918945, -0.09220346063375473, 0.8409239649772644, 0.5963867902755737, -0.5540093779563904, -1.209683895111084, 0.6325421929359436, -0.499651700258255, 1.1998158693313599, 2.0024592876434326, 1.2873731851577759, -0.34542912244796753, 1.3310774564743042, -0.63752681016922, -0.6074292063713074, 0.34643346071243286, -0.22877652943134308, -0.26971137523651123, -1.5529042482376099, 0.2588889002799988, -1.3017667531967163, 1.3319095373153687, 0.8556897640228271, -0.12641143798828125, -1.5239182710647583, 0.2058687061071396, 1.8008297681808472, 0.7814289331436157, 1.5842293500900269, 0.51017165184021, -0.27065524458885193, 0.03007209114730358, 0.9942824840545654, 2.7424116134643555, -0.5712366104125977, 2.859191656112671, -1.2845041751861572, -1.7918510437011719, -0.8773156404495239, 1.1351717710494995, -0.43350347876548767, 0.5485938191413879, 0.3692389130592346, -1.4344921112060547, -1.210176706314087, 1.0863335132598877, 1.0848534107208252, 1.4889675378799438, -0.663033664226532, -0.8074922561645508, -0.37115925550460815], [-2.4457662105560303, -0.06180796027183533, 0.9433236718177795, -0.8207828402519226, 0.7152342200279236, -0.06090924143791199, 1.2475543022155762, -1.2922698259353638, 1.2083137035369873, 1.1801247596740723, 0.6203781366348267, -0.5997412800788879, 0.30784526467323303, -1.4621970653533936, -0.07027245312929153, 1.0920076370239258, -1.8877094984054565, 0.6714289784431458, -0.221905916929245, 0.1881333887577057, -1.5307762622833252, -0.21681088209152222, 0.3260308802127838, -1.6030069589614868, 0.10093662887811661, -0.08603135496377945, -0.6203663945198059, -3.16088604927063, 0.6981764435768127, 0.012494655326008797, -0.3423853814601898, -0.027383215725421906, -1.366347312927246, 0.016583163291215897, 0.483447790145874, 0.602997899055481, -1.3289523124694824, -0.13240735232830048, 0.0035838126204907894, -0.5534303188323975, 0.5470969080924988, 0.6256980895996094, -0.6716078519821167, 0.6494677066802979, 0.029333356767892838, 1.1023441553115845, -1.0828038454055786, -0.0654621496796608, -0.08313976228237152, 0.9285763502120972, 0.4572090804576874, 1.0807044506072998, -0.4241449236869812, -0.1928333044052124, 2.776664972305298, 1.6933379173278809, 0.19782240688800812, 2.680743932723999, 0.20416142046451569, -0.5534151196479797, 1.7752423286437988, -0.22744356095790863, 0.3210166096687317, 0.0025754140224307775], [-0.5186838507652283, 0.517332136631012, -1.5844858884811401, 1.8242815732955933, -0.20606189966201782, -0.18521368503570557, 0.5745282769203186, 0.874958336353302, -0.8610617518424988, 0.10436981916427612, 0.5946172475814819, 0.20210924744606018, -0.577656626701355, 1.467596411705017, 0.6043155193328857, 0.2624417245388031, 0.09028791636228561, 0.38533735275268555, -1.1243951320648193, -0.7027949690818787, -1.0869544744491577, -1.6855134963989258, -0.5907794237136841, 0.5586552619934082, -0.2969200909137726, 0.1954435110092163, 1.2331606149673462, 1.1995952129364014, 0.020838487893342972, 0.0792723298072815, -0.04837634414434433, -0.08028514683246613, 2.502687692642212, -0.8288423418998718, 1.5511237382888794, -1.0692417621612549, 0.31426823139190674, -0.8091357350349426, 1.8724572658538818, -0.8946174383163452, -0.3033958971500397, -0.22004932165145874, -0.9357218742370605, 0.5973391532897949, -0.11764734238386154, 1.1782335042953491, -0.369374543428421, 0.2814478278160095, 0.964221179485321, -0.8234162926673889, 1.2792576551437378, -0.07631585747003555, 0.18791170418262482, -0.28858786821365356, -0.1349557340145111, -0.43595388531684875, 1.1531397104263306, -0.4410403072834015, -0.1210554987192154, 0.16435754299163818, -1.103270173072815, 0.11608806252479553, -0.10132506489753723, -0.44656604528427124], [0.4559701979160309, -0.8686179518699646, 0.061084724962711334, -0.17362827062606812, 0.37435150146484375, 0.26548251509666443, -0.8898806571960449, 1.5845766067504883, 0.37807419896125793, -0.5680215358734131, 0.09753935784101486, 1.9606798887252808, 1.4851362705230713, 1.4985458850860596, -0.2812437117099762, -1.0999947786331177, 0.3566926419734955, -0.8008978366851807, 0.06620161235332489, -0.2886032164096832, 0.0063397386111319065, -0.2394922822713852, -0.05475197359919548, 0.505384624004364, 0.37102875113487244, -0.6491705775260925, 0.6338870525360107, 0.21920840442180634, -1.4085392951965332, 0.30110183358192444, -0.6742554306983948, 0.6342977285385132, -0.6282529234886169, -0.7449384927749634, -0.03806713595986366, -0.47996142506599426, 1.3561275005340576, -0.40497520565986633, 0.22774305939674377, -0.734653651714325, 2.2827165126800537, -1.2034169435501099, 0.05347545072436333, -0.10197672992944717, -0.15667809545993805, -0.7632688283920288, 1.8636016845703125, 0.6764683127403259, 0.07116392254829407, 0.7277584671974182, 0.09739474952220917, -0.7628280520439148, -2.571587324142456, -0.4485107362270355, 0.17922933399677277, 0.2504706382751465, -0.7393057942390442, 1.2985552549362183, -0.21710675954818726, 1.2132289409637451, -0.1500559002161026, 0.0020945658907294273, -0.03326747938990593, 0.519087016582489], [0.3228900730609894, -0.8593879342079163, 0.6617794036865234, 1.2225089073181152, -0.7480682730674744, -1.2308911085128784, 0.05994143709540367, 0.4353022873401642, -0.3204166293144226, -0.7269572615623474, -0.36848706007003784, 0.0883791521191597, -0.18389421701431274, -1.4922178983688354, 0.1931765079498291, -0.6982124447822571, 1.8924260139465332, -1.7480679750442505, -2.2868845462799072, -0.6223365664482117, 1.3402942419052124, 0.8754952549934387, -0.9783267378807068, 0.07755343616008759, 0.5420100688934326, 0.7319329977035522, -0.44508233666419983, -0.9134066700935364, -1.8025552034378052, 0.4472799599170685, 0.4188874363899231, 1.058712124824524, -1.0833861827850342, 0.5248875617980957, -0.7160686254501343, -0.3344080150127411, 1.2989094257354736, 0.2550196647644043, -0.3868897557258606, 1.126731276512146, 0.14549550414085388, -2.0685741901397705, 0.3478788435459137, 1.6825631856918335, -0.44591081142425537, -0.0319126658141613, -0.4475563168525696, 0.801369845867157, 0.4144524335861206, 1.2844569683074951, 1.395151138305664, -0.7613328695297241, -1.1779662370681763, 0.6947259902954102, -2.768852710723877, -0.48805034160614014, 1.2635314464569092, 0.5180907249450684, -0.720120906829834, -0.27421262860298157, -0.21342113614082336, -1.756961703300476, 1.343727946281433, 1.1847724914550781], [1.3278096914291382, 1.0448812246322632, -0.4250622093677521, -0.6977298855781555, 0.9332830905914307, 2.1497130393981934, -0.8549826145172119, 0.6152648329734802, 0.7602731585502625, 0.2477065920829773, -0.08199057728052139, -1.2924120426177979, -1.3788076639175415, 1.2082206010818481, 0.507290780544281, -0.6594592332839966, -1.5650886297225952, -0.9265755414962769, -0.6766035556793213, -0.23079538345336914, -0.2931947410106659, -0.9449940323829651, 0.2789227068424225, 1.0691965818405151, 1.1538763046264648, -0.07563107460737228, -0.5047755837440491, 0.1274086982011795, 0.016931135207414627, 0.906836211681366, 0.3112947940826416, 2.0652964115142822, -0.3837874233722687, -0.4590112864971161, 0.39022091031074524, 1.8551557064056396, -0.29160556197166443, 0.2588514983654022, 0.36810460686683655, -0.010456442832946777, -0.1909642517566681, -2.3711838722229004, -1.9517979621887207, 1.1296409368515015, 1.0161722898483276, 0.11357589811086655, -1.90291428565979, -2.001777172088623, -0.509829580783844, 0.1651730239391327, -1.6004917621612549, 0.9022728800773621, -2.0831871032714844, 0.22013244032859802, -0.7158005237579346, 0.05882538482546806, -0.16757917404174805, -0.2892269492149353, 0.8302673697471619, -0.5129666328430176, 1.3133867979049683, 0.7050530910491943, -1.1196202039718628, 2.4496591091156006], [-0.8023391962051392, 0.20626749098300934, 0.4938201606273651, 0.834542989730835, -0.9471918344497681, -1.1337300539016724, -0.13396631181240082, -0.0264437273144722, -0.4760015308856964, 1.2640405893325806, -1.5690914392471313, 0.2170172929763794, -0.903438925743103, -0.7353507280349731, -0.05699100345373154, 1.093057632446289, -0.7448849678039551, 0.6489841938018799, -0.798150360584259, -1.8203315734863281, 1.0001940727233887, 1.4095224142074585, 2.549175500869751, -0.3602360785007477, 0.26814350485801697, 2.0875184535980225, 0.08879119157791138, -0.07581140100955963, -0.9625719785690308, -0.38474127650260925, -2.0620603561401367, 0.8976895213127136, 0.3226030468940735, 1.043799877166748, 1.5516382455825806, -0.44260168075561523, -0.411037415266037, -0.5764626264572144, 1.085458517074585, -1.3988494873046875, 0.6269262433052063, -1.7494395971298218, -0.5607884526252747, -0.38084501028060913, -0.3720320463180542, 1.368569016456604, 0.5939151048660278, -0.38195154070854187, -0.007441701367497444, -1.0334502458572388, 0.2978605329990387, 0.2742455005645752, -1.0914570093154907, 0.06634914129972458, 0.047871433198451996, 1.041358470916748, 0.00137139274738729, 1.6611828804016113, 1.0812792778015137, 0.3787617087364197, 0.5474417805671692, 2.5975983142852783, -1.7419238090515137, 0.254206120967865], [0.6928125023841858, -0.9663689136505127, -0.9953497648239136, -0.5676820874214172, -0.5955091118812561, 0.25969812273979187, -1.3332229852676392, 0.2924906313419342, -0.004866615403443575, -0.3887077867984772, -0.02840179204940796, -0.5266879796981812, -0.010778995230793953, -1.532274842262268, 0.14294232428073883, -0.6128706932067871, 1.2366787195205688, 0.018984390422701836, 0.012721621431410313, -1.0929769277572632, -1.0819759368896484, 1.2893147468566895, -2.1173477172851562, -1.3326395750045776, 0.2163192331790924, 0.08915290981531143, -0.7387261986732483, 0.708424985408783, 0.3030956983566284, 0.9689813256263733, 1.1454474925994873, -0.6443203687667847, -0.727840006351471, 0.9825707674026489, 0.6210295557975769, 0.5951130390167236, -0.3320356011390686, -1.2296276092529297, -2.029569149017334, -0.385684072971344, 1.2084393501281738, -0.06865614652633667, -0.4310854375362396, 0.7896370887756348, 0.5555201768875122, 0.09971784800291061, 0.8338453769683838, 1.623700499534607, -1.1338032484054565, 0.2735333740711212, 0.6642089486122131, -0.7170248031616211, -0.3992299437522888, 1.7767012119293213, 0.22293737530708313, 0.2584441304206848, -1.147850513458252, -1.1459640264511108, 0.9251961708068848, 0.2002912014722824, 0.18276546895503998, 0.19333042204380035, -0.6512035131454468, -0.3348456919193268], [0.05323700234293938, 0.5591347217559814, -0.26309680938720703, -1.1369469165802002, 0.34392210841178894, -0.36626869440078735, -1.1811964511871338, -0.604424774646759, -0.14962933957576752, -2.250948190689087, 1.4650498628616333, 1.0042246580123901, -0.3081204295158386, -0.3482755720615387, -0.8351917266845703, 0.550808846950531, 0.08780979365110397, 0.35783636569976807, -0.2694678008556366, 0.8951404690742493, -0.9050922393798828, -0.040766410529613495, 0.01998659409582615, -0.8072460293769836, 0.5767450332641602, 1.5927067995071411, -0.37729203701019287, -0.9781373143196106, 0.5458099842071533, -1.7969231605529785, -0.027005674317479134, 1.0551660060882568, 0.9323434233665466, -2.3222339153289795, 1.6448084115982056, 0.7506152391433716, 0.5533108711242676, 0.20714856684207916, 0.3445250689983368, 0.8837210536003113, -0.5850294232368469, 0.23037417232990265, 0.15663985908031464, 0.803235650062561, 1.4516844749450684, 0.36028870940208435, -0.04874734207987785, 1.2332104444503784, -0.1931157112121582, -0.726783275604248, 0.4638090133666992, -0.46207746863365173, -1.0529993772506714, 0.06206965819001198, 1.0319703817367554, 0.6871456503868103, -0.420227974653244, -0.4987255036830902, -0.08743196725845337, 1.4867477416992188, 0.4734687805175781, -0.1710614562034607, 0.6422210335731506, -1.0010395050048828], [-0.8994077444076538, 0.9853976368904114, -0.17798012495040894, 0.2580725848674774, 0.32149749994277954, -0.7321996092796326, 0.19173717498779297, 1.0592756271362305, -0.9811298847198486, -0.27297019958496094, 0.9565866589546204, -0.33167126774787903, 0.13014894723892212, 0.8414945602416992, 0.09271711111068726, -0.5901033282279968, 0.2039569765329361, -0.005113763269037008, 0.056931138038635254, 0.055168043822050095, -0.5242123007774353, 0.2212807983160019, 0.29877546429634094, -0.7150772213935852, -1.4079843759536743, -0.1936500370502472, -0.6241599917411804, 1.824242353439331, 0.1733366847038269, 0.7423948645591736, -0.13247936964035034, -1.4697420597076416, -0.6435667872428894, -1.6179808378219604, 0.7632664442062378, -0.43211641907691956, 0.30125692486763, 2.2717764377593994, 0.0451100654900074, 0.47956588864326477, 1.9192593097686768, 0.654346227645874, 0.1665874719619751, 0.12612996995449066, -0.34336888790130615, -0.2812298536300659, 0.17363432049751282, -0.8079758286476135, 1.068293571472168, -1.1601581573486328, 1.0563052892684937, 0.25907203555107117, -0.15857228636741638, -0.6085914373397827, 0.864898681640625, 1.5595786571502686, -0.20394963026046753, -0.7259882688522339, -0.8158180713653564, 0.1641131192445755, -0.07667411118745804, 0.7555630803108215, 0.7603852152824402, 0.15885429084300995], [0.18820832669734955, 0.02095112018287182, -0.24318991601467133, -0.15603333711624146, 1.7129099369049072, -0.7611191272735596, -0.9683645963668823, 0.029691534116864204, 0.36896929144859314, -0.18741248548030853, 0.6046051383018494, 0.7560038566589355, -0.8020371794700623, 1.0048102140426636, 0.8562765121459961, -1.6312705278396606, 0.4348706603050232, 1.3141380548477173, 0.14388154447078705, 1.5214921236038208, 0.061283666640520096, -1.041006088256836, 0.9922668933868408, 0.0628313198685646, -1.8430365324020386, -0.5210541486740112, 1.4031237363815308, 1.9325487613677979, 0.11445910483598709, 0.7074920535087585, -1.2644785642623901, -0.55689537525177, 0.17753247916698456, -0.791342556476593, -1.0417245626449585, 0.5067647099494934, 0.6263229846954346, 0.10509584844112396, -0.16777074337005615, 0.17791128158569336, -1.780263066291809, -1.3415074348449707, 0.9589616060256958, 2.548375129699707, 0.023683901876211166, -0.14207613468170166, -1.047008752822876, 0.39849618077278137, -0.1578388512134552, 0.2332019954919815, 1.6754080057144165, -0.746673047542572, -1.020192265510559, 0.2701890766620636, -0.5133809447288513, 1.0682564973831177, 0.827446699142456, 0.04805528745055199, 0.22475869953632355, -1.6385600566864014, 1.2109929323196411, -0.03841574117541313, -0.7628070116043091, -0.8194049000740051], [0.4485723674297333, -0.06543561816215515, -1.250236988067627, 1.194366693496704, 1.0231698751449585, 1.5132330656051636, -0.26712167263031006, -0.16221997141838074, -0.21579426527023315, -1.217383623123169, -0.5281493663787842, -1.1482014656066895, -0.5896856784820557, -0.050483401864767075, 0.04838816449046135, -0.46282827854156494, 1.4990805387496948, -1.2637537717819214, 0.5149769186973572, -0.013292325660586357, -0.735598623752594, 2.6288442611694336, 0.2564489245414734, 1.7087477445602417, 0.38963228464126587, -1.4762065410614014, -0.6763784885406494, 0.4664539396762848, -1.7114348411560059, 0.3245890140533447, 2.1138174533843994, 1.4256632328033447, -0.27083876729011536, 0.1255270093679428, -0.8546761274337769, 0.09359115362167358, 0.4694940745830536, 0.17156757414340973, 1.5915453433990479, -0.9338663220405579, -1.7515816688537598, -0.27565187215805054, 0.12134133279323578, -0.2539580166339874, -0.1547384262084961, -0.05357835814356804, -0.24995504319667816, 1.3490772247314453, 1.1072008609771729, -0.5729066133499146, -0.09164173156023026, 0.23502375185489655, -0.8702965974807739, -1.6554358005523682, 0.8631629943847656, 1.690278172492981, 0.7998741865158081, -0.1792822629213333, 1.6746340990066528, -1.466566562652588, -2.0477094650268555, 0.09977424889802933, 0.3819409906864166, -0.9442243576049805], [-1.5602836608886719, 0.8281455636024475, 1.7532823085784912, -0.7167174816131592, -0.2722383439540863, 0.47071772813796997, -0.2100946009159088, 0.04957060143351555, -1.1522973775863647, 1.0790902376174927, 1.9446791410446167, -0.6424611806869507, -0.8628968596458435, -0.17445459961891174, -2.114675283432007, 1.8783597946166992, 0.22337910532951355, 0.1666697859764099, 0.21653059124946594, -0.967158317565918, -1.3829708099365234, -0.7979569435119629, 0.9787920713424683, -0.1279350370168686, -1.5804659128189087, -0.6436313986778259, 0.03383908048272133, -1.138706088066101, 0.7144035696983337, 2.1535868644714355, -0.8565108776092529, -1.2622089385986328, -0.34290993213653564, 0.642242968082428, 1.9478250741958618, -0.4840211868286133, 1.1015247106552124, 1.607191801071167, 0.002066610846668482, 1.2281478643417358, 0.3198915719985962, -0.7026578187942505, -0.5969560742378235, 0.6451132297515869, 0.5080572962760925, -0.27411118149757385, -0.8446376323699951, -1.068898320198059, -0.11191476136445999, -1.2875410318374634, 0.8576706647872925, 0.4285137355327606, 0.6184041500091553, -0.6320573091506958, -0.9807791709899902, -1.4108154773712158, 0.33656635880470276, -1.1210415363311768, -0.18675006926059723, 0.5484127402305603, -1.6528496742248535, 2.045039415359497, 0.3849024176597595, -0.5285258889198303], [0.2683810591697693, 1.7602840662002563, -0.7728594541549683, 0.1768033504486084, 0.5769386887550354, -0.028524091467261314, 0.1723242849111557, -0.8521984815597534, -1.4050610065460205, 1.8980722427368164, -0.14433464407920837, 2.016742467880249, 0.8321250081062317, -0.571778416633606, 0.08670002967119217, -1.0464798212051392, -0.3399456739425659, -0.9597432613372803, 1.6751710176467896, 0.3531453013420105, -1.7296229600906372, 0.19815368950366974, -1.1800793409347534, -0.24891671538352966, -0.19092057645320892, -0.8246937394142151, 0.6729544401168823, -0.645983874797821, 0.1595226526260376, -0.48356449604034424, -1.5927330255508423, -2.5234484672546387, -0.5398781895637512, 1.079124093055725, -0.47930681705474854, -1.9847586154937744, -0.25932633876800537, -0.01685269922018051, -0.4908771812915802, -1.3235875368118286, -0.9994073510169983, 1.0808144807815552, -0.4297438859939575, 0.468821257352829, -0.6836516261100769, -1.5143133401870728, -1.1658248901367188, 0.9273673295974731, -0.789911687374115, 1.3154561519622803, 0.7192064523696899, 0.17268572747707367, -0.8058355450630188, 0.14566856622695923, 2.6581013202667236, -0.7297472357749939, -1.737233281135559, -1.0056147575378418, 0.4603079557418823, 0.6616557240486145, 0.36142903566360474, -0.8714184761047363, 2.949223756790161, 0.9904944896697998], [-0.7328004837036133, -0.5973684191703796, -1.1152989864349365, -0.3209843635559082, -1.9379490613937378, 1.0815966129302979, -0.9052778482437134, 0.7596723437309265, -0.8943527340888977, -1.2599706649780273, 0.07594698667526245, -0.2303367406129837, -0.2392570674419403, 1.4219326972961426, 0.7848470211029053, -1.2633806467056274, 1.0945457220077515, -0.10539299249649048, 1.7455941438674927, -0.7322808504104614, -0.6172069907188416, 0.6861445307731628, -0.01931735873222351, 1.224045991897583, 1.8296493291854858, 0.33563247323036194, -0.09693106263875961, 0.09043648093938828, -1.2685736417770386, 0.7039497494697571, -0.6657208204269409, 0.07480230927467346, 0.2559869587421417, 0.08068535476922989, 0.586118757724762, 0.19608233869075775, -0.33936041593551636, 0.08111611008644104, -0.6649645566940308, -0.0002615133998915553, -0.42781123518943787, -0.7925864458084106, -0.44259655475616455, -0.7820975184440613, 0.13423985242843628, 0.9738863706588745, 1.5823897123336792, -0.42558881640434265, -1.4663385152816772, 1.8743308782577515, 0.615654468536377, -1.7602055072784424, -1.5431385040283203, -1.402603268623352, 0.04254865646362305, -0.48554590344429016, -0.29376402497291565, 0.9022203087806702, 0.4121544361114502, -0.5745101571083069, -0.7196067571640015, -1.0249749422073364, 0.6032640337944031, 1.7149139642715454], [-1.4673023223876953, 1.5221151113510132, -0.026635510846972466, -1.1693649291992188, 0.32592761516571045, 0.3014391362667084, -0.4565153121948242, -0.274466872215271, -0.48525798320770264, 1.8887619972229004, 1.1422758102416992, 1.3981280326843262, 0.31164583563804626, 0.12329567223787308, -0.20234914124011993, 0.8679824471473694, 1.181471586227417, -0.4334677755832672, -0.6170279383659363, 0.12766484916210175, -1.2559573650360107, -1.1259270906448364, -1.3194833993911743, 2.0096592903137207, 0.9068261384963989, 0.010377869009971619, 1.1740849018096924, 1.5257189273834229, 0.009301821701228619, 0.35692501068115234, -1.3859331607818604, 0.47540464997291565, 0.1811995506286621, -1.104823112487793, 1.7833645343780518, -0.022779537364840508, 0.962515115737915, -0.2387925535440445, 0.05010085925459862, 0.25937989354133606, -0.16364747285842896, 0.7447178363800049, 2.366806983947754, -0.12618371844291687, -1.1722627878189087, 0.45053988695144653, -1.031404972076416, -0.9629017114639282, -0.5014632940292358, -2.26971435546875, 0.5683112144470215, -0.6802128553390503, -0.7406947016716003, -2.152094841003418, 0.5080915689468384, -1.487094759941101, -0.21577781438827515, -0.3554767370223999, -0.7277524471282959, -0.7381011843681335, 0.5515188574790955, -1.692151665687561, -0.8796961307525635, -0.5221204161643982], [-1.5653630495071411, 0.004805485252290964, 1.0889145135879517, 1.3809394836425781, -0.9566972255706787, 1.2840858697891235, -0.415688693523407, 0.46445757150650024, -0.9519266486167908, -0.23718971014022827, 1.4471322298049927, -1.2510335445404053, 3.0617012977600098, -1.6492055654525757, -0.5144738554954529, 0.02642393670976162, -1.1954221725463867, 0.06856226921081543, -0.6936342716217041, 0.8753498792648315, -0.31649285554885864, 1.4590924978256226, 1.8182833194732666, 0.1503339260816574, -0.21791449189186096, 2.1153085231781006, -0.8114632964134216, 0.887583315372467, 1.9563572406768799, 0.8916158676147461, -1.3230112791061401, 1.9784613847732544, 0.6303030848503113, -0.5254115462303162, -0.39810773730278015, 0.9759001135826111, -0.696203887462616, 0.39750808477401733, 0.7047371864318848, -0.4036884307861328, -0.5345796942710876, 1.4741169214248657, -0.43269455432891846, 1.1010817289352417, 0.14976303279399872, 0.24225255846977234, 2.338958501815796, 0.8457961678504944, 1.4768848419189453, -0.5999701619148254, 2.205608367919922, 0.8446489572525024, 0.4279315173625946, 0.003943403717130423, 1.9050525426864624, -0.1998850405216217, 0.4181264638900757, 0.02530691958963871, 0.4304240345954895, 0.2349003404378891, 0.09474683552980423, -0.0365903414785862, -0.10463502258062363, 0.08988265693187714], [-0.5088394284248352, 0.24241846799850464, -0.9521441459655762, -1.1708346605300903, 0.5368351340293884, 0.8004239797592163, 0.2739039659500122, 0.18534381687641144, -0.9016581773757935, -1.3301265239715576, -0.10591868311166763, -1.1046571731567383, -2.052279472351074, -1.2796586751937866, -0.2665335237979889, 0.5458608269691467, -1.7827554941177368, 0.9700586795806885, -0.38993555307388306, 1.737533450126648, -0.6505591869354248, 0.7168198823928833, 0.5015312433242798, 0.6793712377548218, 1.700386881828308, 0.4322968125343323, -1.3817991018295288, 0.6496987342834473, -1.2167742252349854, 0.9004126191139221, 0.08811481297016144, -0.16501040756702423, 0.13263724744319916, -1.5401370525360107, 0.08821722865104675, 0.30652734637260437, -0.5181856751441956, -0.12750643491744995, -0.8737974166870117, -0.2475484311580658, -0.23761437833309174, -0.11633431166410446, 0.01120017096400261, 0.03028769977390766, 1.020495891571045, 0.4390898644924164, 1.5713306665420532, 1.1446330547332764, -0.6577968001365662, -0.21471214294433594, 0.26974794268608093, 1.4501360654830933, -0.2526569664478302, -1.1918940544128418, -1.4783992767333984, 1.1391645669937134, -0.9289918541908264, -1.5268707275390625, 3.1770503520965576, -0.7014259696006775, -0.8392229080200195, 0.6626955270767212, 0.21932920813560486, -0.7022587656974792], [0.3438609540462494, 0.6436557173728943, 0.9467040300369263, 1.7075625658035278, -0.008589611388742924, -0.20112349092960358, 0.4784887135028839, 1.7552348375320435, -0.19405347108840942, -1.0326014757156372, 1.4636330604553223, 0.2552320659160614, -0.29845139384269714, -0.42585232853889465, 0.8203288316726685, 0.2345310002565384, 0.011078186333179474, 0.5155613422393799, -1.5834048986434937, -0.1609429568052292, -1.0219281911849976, 0.9077149629592896, 0.054044075310230255, 2.2033276557922363, 0.9699093699455261, -0.20455209910869598, 2.0667009353637695, 1.0938169956207275, -0.08448710292577744, 2.0436532497406006, 1.2073001861572266, 0.2439277321100235, 1.622753381729126, 0.05921907722949982, 0.8742007613182068, -1.563917636871338, 0.19284968078136444, 1.245845913887024, 0.364207923412323, -0.15970629453659058, -0.024261964485049248, 1.1729258298873901, -1.0830600261688232, 0.9554818868637085, 0.13008172810077667, -0.5231940150260925, -0.11061542481184006, -0.07650565356016159, 1.845910668373108, 0.09406289458274841, -2.561109781265259, 1.036840558052063, -0.8413823246955872, 0.07541266828775406, -0.7261099815368652, 0.2358732670545578, 0.9414505958557129, 1.8857611417770386, -1.4116785526275635, -2.505772352218628, 0.6295620799064636, -0.2399066984653473, -1.3167192935943604, -0.09032691270112991], [0.1355714499950409, 0.14179620146751404, 0.5124486684799194, 0.6424591541290283, -0.26133057475090027, -0.34131065011024475, -0.13035395741462708, 0.5815238952636719, 2.2408668994903564, -0.6335433125495911, 2.995161533355713, 0.9497714042663574, 0.3543783128261566, -0.8502188920974731, -0.8575509786605835, -1.1772724390029907, 0.2532397210597992, 0.09860530495643616, -2.134645938873291, 1.8772400617599487, 0.04696028679609299, -1.4975979328155518, -0.25627633929252625, 0.13284477591514587, -0.5444788932800293, -2.0877745151519775, 0.11538798362016678, -0.18346284329891205, 1.7872732877731323, -1.850600242614746, 0.7509773969650269, -0.8076577186584473, 1.5487293004989624, 0.5709378123283386, 0.15700171887874603, 1.687683343887329, 0.2280592918395996, -0.1526404768228531, -0.8272098302841187, -0.5197659730911255, 1.5765198469161987, 0.21561813354492188, 1.943803310394287, 1.1733742952346802, -1.4159436225891113, 0.8144140839576721, 0.09283182770013809, 0.6197530031204224, 0.361486554145813, -0.5867379307746887, -0.7837410569190979, -0.39060601592063904, 1.0449645519256592, 0.1459360420703888, -0.6483834981918335, -0.8268470764160156, -0.07788440585136414, 0.8123283982276917, -0.049668729305267334, 0.7762995362281799, -0.002712318440899253, -0.5234586596488953, -1.2841436862945557, -0.7246200442314148], [-0.3629811108112335, 0.4975737929344177, -0.0013429257087409496, -0.14274679124355316, 0.595867931842804, 0.5901536345481873, 0.4157276749610901, -1.9379284381866455, -2.5631332397460938, 1.1373095512390137, -1.6389390230178833, -0.19934624433517456, -0.4263845682144165, -0.13084834814071655, -0.5112094879150391, 0.1019151583313942, -1.326889991760254, -0.06863895803689957, 0.33002084493637085, 0.4024401605129242, 0.4473345875740051, 1.5847282409667969, -0.295939564704895, 0.29229146242141724, 0.11147522181272507, -0.7854339480400085, 0.013029603287577629, 1.261684536933899, -1.0470271110534668, -0.06191602349281311, -0.009079008363187313, 0.5255480408668518, -1.1283514499664307, -0.6582798957824707, -0.5416892766952515, 1.7266550064086914, -0.02167169563472271, -1.2116605043411255, 0.22044767439365387, 0.33656951785087585, 0.6556538343429565, -0.015535233542323112, -1.5651352405548096, -0.7501357793807983, -1.2405920028686523, -1.266950249671936, 1.0718997716903687, 0.5581529140472412, -0.2986604571342468, 0.6064814925193787, -0.0027505073230713606, 0.3206264078617096, 0.13590091466903687, -0.21136920154094696, -0.294321209192276, 0.7088626623153687, 1.0978533029556274, -0.0767233595252037, -0.2961609959602356, -0.5161328911781311, -2.831542730331421, -0.6761519312858582, -0.8410330414772034, -1.2952247858047485], [0.5144568085670471, 1.7484805583953857, -0.9874358773231506, 0.06521394848823547, -1.1766606569290161, 0.13728681206703186, 0.19005697965621948, -0.1038375049829483, 0.39788326621055603, -0.07896837592124939, 1.7434300184249878, -0.7570106387138367, -0.5372861623764038, 1.2437859773635864, -1.3804534673690796, 1.0084158182144165, 3.186235189437866, -1.0022177696228027, 0.020820865407586098, -1.4362750053405762, -0.28766703605651855, -0.4579404890537262, -0.6598498225212097, 0.1334427148103714, -0.9338165521621704, -0.36204588413238525, 2.449009895324707, -1.4196500778198242, -0.622459352016449, 2.3242311477661133, 1.0412029027938843, -0.9321556687355042, -1.8106719255447388, 0.8787626624107361, 0.6906591653823853, -1.5454881191253662, -0.9824704527854919, 0.6434159874916077, 1.276534914970398, -0.6194380521774292, -0.04213588312268257, -0.21642430126667023, 0.12108708173036575, -1.3052833080291748, 0.6995261311531067, -0.16597068309783936, -0.2974521219730377, -1.2989245653152466, -0.19707809388637543, 1.060102939605713, -0.004891137592494488, -1.654049038887024, -1.0547126531600952, 1.0438367128372192, 1.8532640933990479, -0.07295157760381699, -0.34066012501716614, -0.15965673327445984, -0.47302576899528503, -0.11464780569076538, -0.6445179581642151, -0.49197742342948914, -0.5966141223907471, -0.5343571305274963], [-0.9042201042175293, 0.010551471263170242, -0.27737969160079956, 1.5577702522277832, -0.028013428673148155, -1.1322100162506104, -0.5612869262695312, -0.9361488223075867, -0.41398873925209045, -0.562065064907074, -1.3108512163162231, 0.6834484934806824, -0.4307810068130493, -0.18227633833885193, 0.11000143736600876, -0.7141510248184204, 0.24314694106578827, -1.3269723653793335, -0.3370376229286194, -1.1774548292160034, -0.033553339540958405, 0.7648939490318298, -1.1428695917129517, -0.9588187336921692, 1.294921875, -0.6694405674934387, 0.5699281096458435, 1.379927158355713, -0.8415474891662598, 0.3952704668045044, 1.3006104230880737, 0.17919407784938812, 1.5202524662017822, 0.22925850749015808, -0.1714474856853485, -1.901682734489441, 0.7320346832275391, 0.3461754024028778, -0.6491158604621887, -0.6713771224021912, -0.82810378074646, 0.36385345458984375, 0.43389248847961426, -0.42665979266166687, -1.7851965427398682, 1.6620893478393555, 0.1882297843694687, 1.3042000532150269, 0.15520700812339783, 1.0582962036132812, 0.6992279887199402, -0.4858512580394745, 1.4068478345870972, 0.26230883598327637, -0.8630651831626892, 0.8266353011131287, 0.6806508302688599, 0.896561861038208, -0.4294377267360687, 0.9713060259819031, 0.31360429525375366, -1.5530149936676025, -0.2752155661582947, -1.7538312673568726], [0.33320388197898865, 1.7720741033554077, 1.2528162002563477, -1.1251879930496216, -1.1837406158447266, 0.8485579490661621, -0.4900294244289398, 1.8204421997070312, 0.3938175439834595, 0.022429808974266052, -0.238169327378273, -2.896071434020996, -0.5645374655723572, -1.6098779439926147, 0.48967650532722473, -0.5582370162010193, -0.28274115920066833, 0.30167555809020996, -0.7830982208251953, 0.3649023473262787, -1.594867467880249, 1.6919068098068237, -0.4972308874130249, -0.16639241576194763, 2.590698480606079, -1.254989504814148, 2.5169246196746826, 0.800627589225769, 1.298892617225647, -1.3660824298858643, -1.1203877925872803, 0.040049318224191666, -0.4840072691440582, -1.8809155225753784, -1.1663415431976318, 1.9383535385131836, -0.8470686674118042, 0.03769681975245476, 0.7404600977897644, 0.15987607836723328, -0.60781329870224, -0.2705962061882019, 2.021860361099243, -0.1251608282327652, 1.070538878440857, -0.8484834432601929, 0.4056410789489746, -0.03261605277657509, 0.6655885577201843, -1.0622785091400146, -1.4610044956207275, 0.06703191995620728, -0.027447566390037537, 0.9338600039482117, -0.8204906582832336, -1.3291170597076416, 0.10148579627275467, 1.7332876920700073, 0.036012958735227585, -0.38504758477211, 1.8311446905136108, -1.174011468887329, 0.815760612487793, -0.8328537344932556], [1.654024600982666, 1.5368858575820923, 0.832566499710083, -1.2839727401733398, -0.2636682391166687, 0.9769484400749207, 0.6781467199325562, 1.414144515991211, -0.14269696176052094, -0.954245924949646, -0.22858993709087372, -0.6187891364097595, 1.2078273296356201, -2.012552261352539, 0.9550368189811707, -1.0042909383773804, -0.43543529510498047, 0.614503800868988, 0.8571505546569824, -1.4008897542953491, 1.9177428483963013, -1.4722212553024292, 0.8349295854568481, -0.0020725273061543703, -0.13270890712738037, -0.11012088507413864, -1.0465766191482544, -0.9575805068016052, -0.21796199679374695, 0.38028451800346375, -0.7961642146110535, 1.6428143978118896, -0.7305194735527039, -0.8930944800376892, 0.9331536293029785, -1.5816318988800049, 0.3207910656929016, -0.8141510486602783, 0.7817807197570801, -0.023703569546341896, -0.5400110483169556, -0.47559890151023865, -0.09205856174230576, -1.4574626684188843, -0.4370265305042267, 0.4921092092990875, -0.9409462213516235, 0.0676480084657669, -0.8952999711036682, 1.5313539505004883, 0.37136244773864746, 0.48775729537010193, -0.5774794816970825, -1.3711785078048706, 0.23309758305549622, -1.8392279148101807, 0.7415472269058228, -1.5298784971237183, -0.3532326817512512, -1.0728093385696411, 1.369407057762146, 0.7453040480613708, 0.988364040851593, 0.27812501788139343], [-0.8482487797737122, 0.28001028299331665, 0.1432064324617386, 0.4039936661720276, -0.9451209902763367, -0.04486037418246269, -0.6225103735923767, 0.36066383123397827, -0.8837452530860901, 2.2421445846557617, 1.1952574253082275, -0.02988787554204464, 2.3679418563842773, -0.3815196752548218, 1.3697699308395386, -0.2633614242076874, 2.9526917934417725, 0.8813351988792419, 0.5517677068710327, 0.4816589057445526, -0.4428693354129791, -0.18148671090602875, 1.497053861618042, -0.046266864985227585, -0.61257004737854, -0.4812937378883362, 1.3249174356460571, 0.9383273124694824, -0.24182525277137756, -1.2320538759231567, 0.00901039782911539, 0.8818622827529907, 1.6813511848449707, 0.9989794492721558, 0.5923606753349304, -0.1323232352733612, -1.9266289472579956, 0.4148634970188141, 0.671952486038208, 0.40175482630729675, 0.5513256192207336, 0.9493369460105896, 1.3577512502670288, -0.1917872577905655, 1.2788848876953125, 1.8344160318374634, -2.1205058097839355, 0.07851222902536392, -0.530276358127594, 1.0518450736999512, 1.5798308849334717, 1.0428541898727417, -0.05100178346037865, -1.3814775943756104, 0.5442612767219543, -0.6926948428153992, -0.27203017473220825, -0.3610125482082367, -1.302903652191162, -0.5854063034057617, -1.2651066780090332, 1.7395437955856323, 0.4348996877670288, 0.38511526584625244], [-2.632695198059082, 0.5409639477729797, 0.06401141732931137, -0.04995984584093094, 0.3807010054588318, -0.5720102787017822, -0.9532997012138367, 0.3558782935142517, -0.3177790939807892, 0.19888024032115936, 0.3409346640110016, 1.4840120077133179, -0.5872305631637573, 0.953946590423584, -0.004412026144564152, -0.45043253898620605, 0.6978078484535217, 0.5816656351089478, -0.07109076529741287, -0.3935486376285553, 0.6468886137008667, 0.5477946400642395, 0.8153837323188782, 0.5256317853927612, 0.34104886651039124, 0.1305147409439087, 0.5224930644035339, 1.4448425769805908, 1.7176004648208618, -0.22047755122184753, 0.47002020478248596, -0.41352930665016174, 0.58670574426651, -1.7400821447372437, -2.000379800796509, 0.19301439821720123, -1.0752995014190674, -1.2329819202423096, 1.5600162744522095, 1.3772188425064087, -0.3833700716495514, -0.06138171628117561, -0.6641845107078552, 0.9412396550178528, -0.9819749593734741, 0.058570895344018936, -0.7419354915618896, 0.4531744122505188, 0.696201503276825, 0.33424606919288635, 0.34194499254226685, -0.6515266299247742, 0.4433274567127228, 0.18564732372760773, 0.5380769371986389, -0.7001976370811462, -0.0626886785030365, 1.143702745437622, 0.4566478431224823, -1.7939705848693848, -0.00802661757916212, -0.15702278912067413, -0.5116978883743286, 0.13020864129066467], [-2.0820202827453613, 0.04100894182920456, 0.8526857495307922, -0.838201105594635, 0.7415248155593872, 1.290780782699585, 1.340327262878418, 0.1270648092031479, -0.4197993874549866, 1.3045860528945923, -1.705775260925293, 0.4164331257343292, 0.18385548889636993, -0.30424097180366516, -0.9985790848731995, 0.18438410758972168, -0.48272591829299927, 0.12822484970092773, -0.03827979043126106, -0.44902241230010986, -0.9630494713783264, 0.7343745231628418, 0.4307575523853302, 1.263076663017273, -0.37414002418518066, 0.326486736536026, 1.1910839080810547, -0.39713233709335327, 0.044106751680374146, 0.3643365800380707, -1.1153713464736938, -0.6854161024093628, -0.18909898400306702, -1.4094499349594116, -0.5688686370849609, -0.11615359038114548, 0.9333507418632507, 0.6396010518074036, 0.41710567474365234, -0.8400843143463135, -1.322571873664856, -2.806709051132202, 0.17267513275146484, 0.803975522518158, 0.815352201461792, -0.2106798142194748, 0.32551631331443787, -0.7424641847610474, -2.3274013996124268, -0.39018529653549194, 0.41335955262184143, -0.7765375971794128, -0.8729466199874878, -0.06291736662387848, -0.44677045941352844, 1.1439135074615479, 0.8517506122589111, 0.39312705397605896, -0.8150008916854858, -1.2059767246246338, -0.6383183598518372, 1.349952220916748, 1.0461235046386719, -0.7449228763580322], [-0.5347731113433838, -0.3814264237880707, 1.2350856065750122, -0.6621562242507935, 0.038693830370903015, 0.48802822828292847, -1.7331585884094238, -1.251062035560608, -1.1122158765792847, 0.941100001335144, 0.7687039375305176, 1.1421146392822266, 0.9610121846199036, -0.057845890522003174, -0.3122171461582184, 2.010019540786743, -0.10746730118989944, 1.1762317419052124, 0.7272300720214844, -0.8763648867607117, -0.8151827454566956, 0.03771377354860306, 2.7076032161712646, 1.146276831626892, 2.0933783054351807, -0.7077325582504272, -0.5897672772407532, -0.14985667169094086, 0.015227283351123333, -0.8631095290184021, 0.7569831609725952, 0.7441086173057556, -0.2262798696756363, -0.712256133556366, 0.8866595029830933, 0.23914849758148193, -1.5841370820999146, -0.357179194688797, 0.03806358575820923, -0.43354812264442444, -0.5648476481437683, -0.8675788640975952, 0.6964117884635925, -0.9186943769454956, 0.28170061111450195, -0.5618027448654175, 0.28777724504470825, -1.0420516729354858, 0.26945891976356506, -0.3630976378917694, 0.6776520013809204, 0.16763515770435333, 0.5581192970275879, -1.0471216440200806, 0.11835265159606934, 0.20043009519577026, -0.8120314478874207, -2.3457529544830322, 0.057723525911569595, 0.13015824556350708, -0.13297134637832642, -0.04324069619178772, 1.448441982269287, -0.9642425775527954], [0.5735388398170471, 0.37473294138908386, -1.601836085319519, -0.7606503367424011, 1.4820740222930908, -0.8193193078041077, -1.3414227962493896, 0.9293043613433838, -0.7054646611213684, 0.5952792167663574, 0.2720620334148407, 0.594616174697876, 0.05959763005375862, 1.2883800268173218, -0.7206231951713562, 0.9629166722297668, -0.24372901022434235, 0.7628931403160095, -0.2028166949748993, -1.200020432472229, 0.009248674847185612, -0.3970572054386139, -0.6060107350349426, -0.18420201539993286, 0.38387128710746765, 0.9866289496421814, 0.708582878112793, 0.9196200966835022, 0.9327735304832458, 0.41079288721084595, -0.4412466585636139, 0.42135778069496155, -1.039285659790039, 0.5564766526222229, -0.9012961983680725, 0.6195715665817261, -1.882709264755249, 0.188408762216568, -1.3378846645355225, -0.4735226333141327, 0.5234178304672241, 0.4035622775554657, -0.09572558104991913, 1.2306265830993652, 1.247967004776001, -1.806983232498169, -0.49451324343681335, 0.9695226550102234, -0.18435119092464447, -0.06241699680685997, 1.0167585611343384, 0.3083273470401764, 2.3983545303344727, -0.010037694126367569, 0.6846660375595093, 1.1735774278640747, 0.16249729692935944, -0.4923171103000641, -0.9665382504463196, 0.4282297194004059, -0.10605672746896744, 1.2460064888000488, 0.14302432537078857, 0.24413880705833435], [1.6966711282730103, -1.657365083694458, -1.1130707263946533, 1.1087332963943481, 1.183754324913025, 1.0051995515823364, -1.1006635427474976, 0.7413245439529419, 0.40749284625053406, -0.12212631106376648, -0.8235224485397339, 1.3916821479797363, 0.13853971660137177, -0.04086943343281746, -0.16769327223300934, 0.6390882730484009, -0.07077118009328842, 0.15143710374832153, 1.3156243562698364, -0.36621975898742676, -0.5826055407524109, 1.221962809562683, 1.4023048877716064, -1.199852705001831, 1.2525407075881958, -0.17799516022205353, 2.176177978515625, -0.7760747671127319, -0.6170791387557983, 0.3771825432777405, -0.31704339385032654, 1.5342273712158203, -0.28241288661956787, -0.11978869140148163, 0.8735467195510864, -1.2278881072998047, 0.5746042728424072, -1.6568777561187744, 0.27624937891960144, 1.1955968141555786, 0.8272745609283447, 0.850917637348175, 1.7386330366134644, -0.5243723392486572, 1.958943247795105, 0.0037511298432946205, -1.175503134727478, 1.0367732048034668, -0.49207964539527893, -0.39454537630081177, 0.7561478614807129, 0.2853092551231384, -0.5827335715293884, -0.3593246042728424, -0.24444791674613953, 0.863259494304657, 0.4240312874317169, 1.9688247442245483, 1.2618777751922607, 1.9868953227996826, -0.3309359848499298, -1.7642368078231812, 0.26378971338272095, 1.256899356842041], [-1.3960742950439453, 1.1558598279953003, -0.8095535039901733, 0.11133073270320892, 1.6702648401260376, 1.2040210962295532, -1.5497699975967407, -0.3243376612663269, 0.7797338962554932, 0.6752640604972839, 2.1596460342407227, -0.020849736407399178, 0.40095534920692444, -0.17472627758979797, -1.3212223052978516, 0.6154279708862305, -0.005394160281866789, -0.0647570937871933, -0.5336366891860962, -0.8513162136077881, 2.2687509059906006, 1.5765647888183594, -2.347930431365967, 0.3055943548679352, 0.35946112871170044, 0.26273447275161743, 0.7360596060752869, 1.2471636533737183, -0.3038617968559265, -1.437694787979126, -1.3171279430389404, 1.512297511100769, 0.575497567653656, 0.6082251071929932, -0.17996099591255188, 1.6528092622756958, 0.2551531195640564, -0.6974638104438782, 0.7534221410751343, -0.08998309820890427, 0.8366634845733643, -0.02600858546793461, -0.07402783632278442, -1.3419758081436157, 0.5732945203781128, -0.8477500677108765, 0.2734104096889496, 0.220319926738739, 0.4774256944656372, -0.22094030678272247, -0.9726256728172302, 1.2331759929656982, 0.9037920236587524, 0.021023716777563095, -0.4207145869731903, -0.650705873966217, 0.6941226720809937, -0.28203362226486206, -1.3625901937484741, -1.8933351039886475, -0.5751640796661377, -1.113928198814392, 0.9867660999298096, 0.5861059427261353], [0.5627792477607727, 1.3818811178207397, 0.6468928456306458, -0.7159239649772644, 1.294478178024292, 0.5249150991439819, -0.1804475039243698, -0.1421670764684677, 0.7537264227867126, 1.077289342880249, 0.06918301433324814, 1.4190492630004883, 0.6068430542945862, -0.48938220739364624, -0.02724572829902172, -0.8465855717658997, 0.1372208446264267, 1.2076456546783447, -1.282179594039917, 0.6285771727561951, 1.35248601436615, 0.4645381271839142, -1.0358952283859253, 0.6504817605018616, 0.15737879276275635, -1.3767614364624023, 0.5382622480392456, 0.9289586544036865, 2.0394816398620605, -0.6370421051979065, 2.0856149196624756, -0.5568260550498962, -0.5044042468070984, -1.939571738243103, -0.30657482147216797, -0.5071814656257629, 1.0474090576171875, 1.263993263244629, 1.1409127712249756, 0.23780223727226257, -1.5317797660827637, 1.050894856452942, 0.2736961245536804, 0.01676096022129059, 1.7723760604858398, -0.02736741304397583, -0.14384767413139343, 0.516818106174469, 0.5569167137145996, 0.189638152718544, -0.2154087871313095, 0.2636285126209259, 0.5919983386993408, -0.7286537885665894, 1.4357097148895264, 0.8712601065635681, 0.28295326232910156, -0.6575936079025269, -1.2196711301803589, 0.49435943365097046, -0.04276967793703079, 2.0840604305267334, 0.3970201313495636, 1.045681357383728], [1.4277780055999756, 0.45796164870262146, 2.050654649734497, -0.4303101897239685, -1.032843828201294, -1.3081605434417725, -0.15781304240226746, -0.5905329585075378, -1.3198293447494507, -0.7977198958396912, 2.1938984394073486, 0.796523928642273, 0.1917230784893036, 0.47444137930870056, -0.05863919481635094, -0.7567557692527771, 0.42438122630119324, 0.8273571729660034, -0.8507793545722961, -0.1223885789513588, 1.2338944673538208, 0.8018617033958435, -0.9375429153442383, 1.2690144777297974, 0.33042460680007935, 1.0013056993484497, 1.0164514780044556, -0.4542704224586487, 0.7195011377334595, 1.3235043287277222, -1.8212120532989502, 0.8561845421791077, -1.9664353132247925, -0.002112023066729307, -0.10383386164903641, 0.40258827805519104, 1.0191022157669067, -1.3567352294921875, 0.32873421907424927, 0.8793315887451172, -1.7170023918151855, -0.4717392921447754, 0.24972420930862427, 0.018947167322039604, 0.5087318420410156, -0.7783183455467224, 0.6762726902961731, -0.9814937710762024, -2.295513868331909, -0.38349485397338867, 1.0530266761779785, -1.3032746315002441, -1.4733502864837646, 1.6337335109710693, -2.2192347049713135, 0.7218294739723206, -0.8995405435562134, 0.24717754125595093, 0.7477686405181885, -1.8095561265945435, -0.5583372116088867, 0.7700580358505249, -0.8961204886436462, 0.6709347367286682], [0.10452277958393097, 1.7109630107879639, -0.38752442598342896, 1.1903198957443237, -1.0827854871749878, -0.8811695575714111, -1.160591959953308, 1.4863383769989014, -0.2985304296016693, -0.8985070586204529, 1.5078905820846558, 0.266409695148468, 1.5603983402252197, -0.5230238437652588, -0.1866784691810608, 1.4213272333145142, 0.18790635466575623, 0.38610199093818665, -0.5607926249504089, -0.13243363797664642, 0.22516268491744995, 0.6842875480651855, 0.007657863199710846, -1.106572151184082, -0.02394767850637436, -1.002095103263855, -0.23685823380947113, -2.8579213619232178, 1.3533447980880737, -1.0006660223007202, -0.4828302264213562, -0.5050649642944336, 0.6302633881568909, -0.26107296347618103, -0.5815623998641968, -1.7615306377410889, 0.8402713537216187, -1.6553007364273071, 0.0817314088344574, -0.4738026559352875, 0.1869470477104187, -1.1141144037246704, 1.744234561920166, 0.15980830788612366, -0.39387792348861694, -0.1244146004319191, -0.787395715713501, -1.7435866594314575, 0.2725795805454254, -1.0332605838775635, 0.7603990435600281, -0.2691841721534729, 0.7974588871002197, 0.4463396668434143, -1.0185129642486572, -0.7893991470336914, 0.4557999074459076, -0.32809582352638245, -0.40777087211608887, -0.882818877696991, 1.921808123588562, 0.058832865208387375, 0.4954540431499481, 0.21217714250087738], [1.113952398300171, -0.44030556082725525, -0.37208887934684753, -0.3179463744163513, -1.124802589416504, 1.4889094829559326, 1.4381721019744873, -0.9147716760635376, -0.7900915145874023, -0.7952115535736084, -0.6200568079948425, 0.22913579642772675, -0.815115213394165, -1.7512129545211792, 1.1969337463378906, 0.5009135603904724, -1.6591358184814453, 1.0239686965942383, 1.0609034299850464, 0.20244470238685608, 0.23498769104480743, -0.7555624842643738, 0.47127100825309753, 1.1294981241226196, 0.14339828491210938, 0.9062420725822449, -1.3997641801834106, -0.8726619482040405, -0.21238262951374054, -0.7872068285942078, -0.2450627088546753, -0.834014892578125, -0.9020562171936035, 0.9226900935173035, 0.10274305939674377, -0.9590861201286316, 0.42201098799705505, -0.6476867198944092, -0.9950173497200012, -1.1009809970855713, 1.227168083190918, -0.7808517217636108, -1.1821953058242798, -1.4063929319381714, 0.6594264507293701, 0.359836220741272, 0.6508097648620605, -0.0648287907242775, 0.4311760663986206, 1.6843538284301758, 0.1351100653409958, -0.0761333703994751, 0.9573599100112915, -1.0780575275421143, 0.46431201696395874, 0.20136983692646027, 0.17718255519866943, 0.5908383131027222, 0.7463293671607971, 0.3846406638622284, 0.10525355488061905, 1.0416933298110962, -0.7208669781684875, -0.2875104248523712], [-0.06686951965093613, -0.5837838649749756, 0.20863574743270874, 0.1592748910188675, -1.5212836265563965, 1.056588888168335, 0.6253430247306824, 0.8456527590751648, -0.35468238592147827, -1.1928272247314453, 0.30256178975105286, -1.491262435913086, -1.2695168256759644, -0.14433254301548004, -2.5681369304656982, 0.654847264289856, -0.7388754487037659, -0.6056039929389954, -0.10785873979330063, 0.4686146378517151, -1.1971076726913452, 1.3212865591049194, -1.046413779258728, 0.7656695246696472, -0.2965161204338074, 0.6538439393043518, -0.7910274863243103, 0.24591456353664398, -0.8613685965538025, -1.1501559019088745, -0.8778127431869507, 0.3352131247520447, 0.7747799158096313, 0.0010020160116255283, 0.6952851414680481, -1.286453127861023, 0.9664139151573181, -0.6790744066238403, -0.7466079592704773, 0.714483380317688, -0.09443633258342743, -0.3205567002296448, -1.2347453832626343, 1.1240330934524536, 0.5727078318595886, 0.24397747218608856, -0.18853521347045898, 0.009536885656416416, 0.14794921875, 0.374436616897583, 0.14589332044124603, 0.37833791971206665, 0.55018550157547, 2.5089762210845947, 1.2663898468017578, 0.3660641312599182, -1.0682475566864014, 0.30898791551589966, -0.6444072723388672, -0.6952475905418396, 0.7014667391777039, -1.3841538429260254, 0.54688560962677, -0.9255220890045166], [1.3906956911087036, 1.4924482107162476, 0.9468843340873718, -1.0856751203536987, -0.27755799889564514, -0.7603863477706909, 1.224657654762268, 0.1498815268278122, -2.661808729171753, -1.4910892248153687, -1.010598063468933, -0.26606622338294983, -0.5626088976860046, 0.11317852884531021, -0.09829756617546082, 0.6017816066741943, 0.5777137875556946, 1.6772211790084839, 1.9326494932174683, 2.1247754096984863, 1.8631376028060913, -0.9101490378379822, -1.6064367294311523, 0.5299452543258667, 0.18969981372356415, -0.7216814756393433, -2.1004817485809326, 0.5646388530731201, -0.02949080988764763, 0.9082649350166321, 0.31254827976226807, 1.8314547538757324, -0.4309203624725342, 0.1163298636674881, 0.09753727167844772, 0.9066964983940125, 0.19102546572685242, 1.6509547233581543, 0.8222209811210632, 0.38379836082458496, 1.2073569297790527, -1.2487033605575562, 0.5007452368736267, 0.20564401149749756, -0.7909513115882874, -0.2998007833957672, 0.6795732975006104, -0.5650932192802429, -0.01920905150473118, -0.35159772634506226, 0.49653568863868713, -1.629093050956726, 0.7941891551017761, -0.5333626866340637, -1.3965781927108765, 0.567098081111908, -1.4226821660995483, 1.1524055004119873, 0.5302225351333618, 0.7753446102142334, -0.707068920135498, 1.595586895942688, -0.48845911026000977, 0.12960799038410187], [0.552821934223175, -0.42295700311660767, 2.089064598083496, 0.057282064110040665, -0.5566138029098511, -0.8303237557411194, -1.6372463703155518, 0.31583189964294434, -0.2611987888813019, -0.6229043006896973, 0.6767932176589966, 0.9754725098609924, 0.28371742367744446, -1.4898992776870728, 0.29123497009277344, -0.4056055247783661, 0.06117631494998932, 1.4787907600402832, 2.1572153568267822, -0.5349280834197998, 0.07775387912988663, -0.35426485538482666, 0.304127037525177, -1.8051272630691528, 0.3802456557750702, -0.9889448881149292, -1.1659059524536133, 0.6580734252929688, 0.7840456366539001, -0.5069587826728821, 0.7717868685722351, -1.9172979593276978, -0.37736114859580994, 1.5203288793563843, 0.722532331943512, 0.7976427674293518, -1.1319524049758911, -1.0771101713180542, -0.6880845427513123, -0.07935401052236557, -0.6167700886726379, 0.5401305556297302, 1.1730269193649292, -0.5779480934143066, 1.5804574489593506, 0.44583818316459656, -0.7510301470756531, -1.4066578149795532, 0.3188175559043884, 0.30230310559272766, -0.7682546377182007, 0.47441500425338745, 0.36808159947395325, -0.6524838805198669, -0.18648867309093475, 0.15841132402420044, -0.17173753678798676, -1.3876250982284546, -1.0085296630859375, 1.0981374979019165, -0.6072736978530884, 0.7464823126792908, 0.8595427870750427, -0.4632927179336548], [2.0129778385162354, 1.1016976833343506, -1.2285754680633545, 1.6888401508331299, -0.24529510736465454, 0.024341218173503876, -0.16098862886428833, -1.471697211265564, 0.10556800663471222, 0.4365686774253845, -0.08747469633817673, -1.195595145225525, 1.4946091175079346, -2.4966001510620117, -0.41322508454322815, -0.03370010107755661, 1.9227125644683838, 0.14756804704666138, 0.8100153803825378, 0.4002739489078522, -0.7369877099990845, 0.4887143075466156, 0.8916969895362854, 0.6788318157196045, -1.1135309934616089, 1.4133280515670776, 0.1021890714764595, -0.48741158843040466, -1.3128166198730469, 0.44334304332733154, 1.3180445432662964, 0.24401983618736267, 0.31043094396591187, 0.1647697538137436, -0.1725464165210724, -1.2745002508163452, 0.7033622860908508, 0.7126272916793823, -0.7139950394630432, 0.7708525061607361, 0.72552490234375, -1.5214635133743286, -0.28585317730903625, 0.9923873543739319, -1.353481411933899, 1.7019603252410889, 0.6213769316673279, -0.8646239638328552, 0.27978238463401794, 0.2265085130929947, -0.5760381817817688, 0.6385385394096375, 0.052016451954841614, -0.3451295793056488, -1.1102197170257568, -0.5340981483459473, -0.17505869269371033, 1.028337001800537, 0.869013786315918, -0.5393466949462891, 0.04205729067325592, -1.2185988426208496, -0.2415771186351776, 2.6114401817321777], [0.7513322830200195, -0.9056969285011292, -1.6986557245254517, -0.7414001822471619, -0.1395488828420639, -1.4162139892578125, 0.17042435705661774, -1.135567307472229, 0.6035904884338379, -0.1557467132806778, -0.14709414541721344, -0.7002960443496704, 0.8165602087974548, 1.1210473775863647, 1.6294740438461304, 0.46519193053245544, 0.4999135434627533, -0.5121997594833374, -0.5413545370101929, -0.34679412841796875, -0.6685091257095337, -2.2676570415496826, -0.345121830701828, -0.15180683135986328, -0.39854228496551514, 0.8988651633262634, 1.2999539375305176, -0.6579527258872986, 1.1433045864105225, 1.2491086721420288, -0.9113154411315918, -0.1269284337759018, -0.3867562413215637, 0.8627190589904785, -0.6283418536186218, 0.2934935688972473, 0.6940237283706665, 0.9247086644172668, 0.9884183406829834, 1.1505523920059204, -0.12119433283805847, -0.7766507863998413, 1.387234091758728, 0.32161590456962585, -0.8219866156578064, -0.0052973986603319645, 2.007871389389038, -1.3657329082489014, -0.6672674417495728, -1.0481687784194946, 1.2218880653381348, 0.7573264241218567, 0.5358871817588806, -0.05039310082793236, 0.5089477300643921, -0.3842253088951111, 0.11627514660358429, -0.3852185010910034, 0.03424558788537979, -2.1588053703308105, 0.48955103754997253, 0.47482332587242126, 1.4983160495758057, 0.28308939933776855], [0.4955582618713379, 1.1294904947280884, 0.6430730223655701, 0.16388651728630066, -0.6091916561126709, 0.8204978704452515, 2.503141164779663, -1.0344558954238892, 1.5130475759506226, 1.61299467086792, -0.012822034768760204, -0.3899449110031128, 0.12488735467195511, -1.4721652269363403, 0.30996760725975037, -2.102670192718506, -0.8120834231376648, 0.4499457776546478, 0.0057775420136749744, -1.3666855096817017, -0.24205070734024048, 0.813768208026886, -0.31154996156692505, -1.7738163471221924, 1.4154938459396362, -0.22874192893505096, -0.23516744375228882, -0.19115257263183594, 1.0079569816589355, 0.8650625348091125, -0.9624125361442566, -0.8980762362480164, -0.11804517358541489, -0.7886472344398499, 0.770574152469635, 0.9630460739135742, 0.1448492705821991, 2.219447374343872, -1.6796596050262451, -1.894059419631958, 0.121049664914608, 0.19871874153614044, 0.8348866105079651, -1.8035207986831665, 0.3886614739894867, 1.147547960281372, -1.2219573259353638, -0.6075936555862427, 1.4947288036346436, -1.259173035621643, -1.00810968875885, -0.25760406255722046, 0.6630613207817078, -0.8695697784423828, 0.27424484491348267, -0.8296306133270264, -0.05799521878361702, 0.515083909034729, -0.24957266449928284, 1.2934397459030151, 1.6295297145843506, -0.5044779777526855, -0.03193432837724686, 1.3068073987960815], [-0.4182208180427551, -0.14034724235534668, 0.6186515092849731, -0.3568520247936249, 1.1495482921600342, 1.1151418685913086, -0.9417270421981812, -0.38251522183418274, -0.3106653094291687, -0.5824573040008545, -1.1655423641204834, -0.684026837348938, 2.427442789077759, -0.2734311521053314, -0.740669310092926, 0.4584190249443054, -2.6121723651885986, -0.4905577301979065, 0.9375671148300171, -0.639946699142456, 0.31512463092803955, -0.9733035564422607, -1.7129930257797241, 1.038091778755188, -0.7158247232437134, 1.108515977859497, 1.1747641563415527, 0.5765145421028137, -1.63741934299469, 1.1827863454818726, 2.1040897369384766, -0.398519366979599, 0.023808948695659637, -0.8012300729751587, 0.884407103061676, -0.4411909878253937, 0.5670541524887085, -0.6170486211776733, -1.1516733169555664, 0.025016319006681442, -0.6842013597488403, 0.05273551121354103, 0.7629727125167847, -0.1390582025051117, -0.7512024641036987, 1.1328483819961548, 0.7487834095954895, -0.6528141498565674, -0.29670462012290955, 1.122300148010254, 0.18682150542736053, 0.5906379818916321, -0.3386070728302002, 2.627149820327759, -0.1716948002576828, 0.5052282214164734, 0.7024074792861938, -0.7211119532585144, 1.5658190250396729, 0.6909517645835876, 0.07840912789106369, 0.5208868980407715, 1.081073522567749, -0.06780128926038742], [-1.7772793769836426, 0.5139992237091064, 0.05342157930135727, 0.8463438749313354, 0.8282654881477356, 0.4966234266757965, -0.9272811412811279, -0.2075200080871582, -1.6991608142852783, -1.9771910905838013, -0.850207507610321, 1.0203121900558472, 0.29641664028167725, -0.23618926107883453, 1.0135715007781982, 1.23958420753479, 1.3498497009277344, -0.9219691753387451, 0.2757839560508728, 0.5328432321548462, 2.0138027667999268, 0.4207116961479187, 1.162919521331787, 0.08760664612054825, -0.6129361987113953, 0.9444833993911743, 0.8583475947380066, 1.5547573566436768, 1.205552101135254, -0.9595672488212585, 1.3464807271957397, -0.6192066073417664, 0.17235630750656128, -0.4124571979045868, -0.6445180177688599, 0.25414642691612244, 1.5241022109985352, 0.2441813051700592, -1.4141857624053955, 0.3752168118953705, 1.125626564025879, -1.385464072227478, -0.6852383613586426, -0.6771084070205688, 1.1924396753311157, 0.09668850898742676, 1.3957784175872803, -0.07481256127357483, -0.5848400592803955, 0.6374312043190002, 0.3569561839103699, -0.8062191009521484, -0.5516774654388428, -0.03414415940642357, -0.39534875750541687, -0.5215046405792236, 2.466306686401367, 1.8513120412826538, 2.0916266441345215, -0.4230639934539795, 2.1079678535461426, 0.8793771266937256, -0.19039234519004822, -1.622323989868164], [-1.4165582656860352, -0.05802397057414055, -1.2440848350524902, 0.11467728763818741, -0.005346913356333971, -0.08224157989025116, 0.6358177065849304, -1.3004511594772339, 1.136110544204712, -0.4977158308029175, -0.4277031123638153, 0.460840106010437, 0.06533748656511307, -0.07018427550792694, 0.9513471722602844, -0.5529805421829224, -2.395596742630005, -0.12536446750164032, -0.19682006537914276, 0.9624678492546082, 0.010668512433767319, -0.06142403185367584, -0.14415842294692993, 0.37874317169189453, -0.9009551405906677, 0.034087155014276505, 1.8016514778137207, -0.266044944524765, 0.8380311727523804, 0.9655921459197998, -0.9145910143852234, -1.9138206243515015, 0.4431403875350952, 2.22430419921875, 0.008778003975749016, 0.4032595753669739, -0.7604554295539856, -0.49245333671569824, -0.2171047478914261, -0.43944305181503296, -0.20042340457439423, -0.5045763850212097, -0.28540071845054626, 0.690582811832428, 0.142875537276268, 0.7273674607276917, -0.25420477986335754, 1.2900540828704834, -1.0677684545516968, 1.543978214263916, -0.7241043448448181, -1.2559798955917358, 0.5531309247016907, -0.8809900879859924, -1.9138845205307007, -1.8779011964797974, 0.3032284379005432, -0.26127275824546814, 0.1952284276485443, -0.7444229125976562, 0.2961858808994293, 1.4184201955795288, -1.0578166246414185, 0.6816287636756897], [0.38646525144577026, 1.0149141550064087, -0.27074068784713745, 2.280158519744873, 0.63633131980896, 0.2610882818698883, -0.21801690757274628, -0.5575251579284668, -1.5111467838287354, 0.3076949715614319, -1.604264736175537, -1.3293448686599731, -0.4780293405056, -0.4365849196910858, -0.11412075161933899, -1.5032702684402466, -1.0374163389205933, 0.6655272245407104, -0.6869649291038513, 1.064350962638855, 1.2012269496917725, 0.6232191324234009, 0.7585224509239197, -0.09416953474283218, 1.2320183515548706, -0.08315480500459671, -0.559657633304596, -0.7144620418548584, 0.4468356668949127, 0.6505120992660522, -0.6158883571624756, -1.8860950469970703, 0.9785823225975037, 0.8503643274307251, 0.8088652491569519, 0.7646925449371338, -0.6514284610748291, 0.11716365814208984, 0.8640218377113342, -0.7149126529693604, -0.24164101481437683, 1.4399607181549072, -1.2310739755630493, -1.1835966110229492, -0.45346856117248535, 1.1961838006973267, -0.14389248192310333, -0.05030215531587601, 0.5226831436157227, -0.3299899101257324, 0.4476339519023895, -1.113020658493042, -0.7075902223587036, -0.8711868524551392, 0.285720556974411, 2.6691300868988037, 0.6255404949188232, -0.5960742831230164, -2.0348165035247803, -0.2629334628582001, 0.39147844910621643, -0.5648026466369629, 0.01092151366174221, 0.5612869262695312], [0.4529109001159668, -0.3637489974498749, 0.36640405654907227, 0.2861405909061432, -1.2952884435653687, 0.9481825828552246, 0.19974792003631592, 1.3498682975769043, -0.5160651803016663, -0.0832185298204422, -0.9002315402030945, 1.6819857358932495, 0.5774943828582764, -1.1677806377410889, 1.0684329271316528, 0.2590074837207794, -1.850629448890686, 1.191599726676941, 0.9022508859634399, -0.06705282628536224, 1.1807546615600586, -0.5716021060943604, -1.6569774150848389, 0.7724812030792236, -0.5353378057479858, -1.0555812120437622, 0.9855092763900757, -0.10666362941265106, -0.5123329758644104, 1.0771552324295044, 1.9979397058486938, -0.3394501805305481, 0.22913525998592377, 0.27947619557380676, -1.2042615413665771, 0.567594051361084, -0.787666380405426, 1.7607879638671875, -0.995938777923584, 0.25795087218284607, 0.33883288502693176, 0.7437608242034912, 0.7969427108764648, 1.6235826015472412, 0.09657857567071915, -0.6947653293609619, 1.2489043474197388, -0.3221375048160553, 0.1563318520784378, -0.04853082075715065, -0.5085383057594299, 0.3685430586338043, 0.23901590704917908, -2.3085076808929443, 0.18367953598499298, -0.841438889503479, 0.2524309456348419, 0.2240126132965088, 0.11227406561374664, -0.7879511117935181, 0.08488903939723969, -0.20798861980438232, -0.07839512825012207, 0.7673499584197998], [0.2145431488752365, -0.7023177146911621, 0.6085174679756165, 0.9345865845680237, -0.5029765963554382, -2.4463579654693604, -0.41774171590805054, 0.7786872386932373, -0.5986257195472717, -0.34051376581192017, 1.0521332025527954, 0.2609499990940094, 1.9174065589904785, 0.8891857266426086, -0.5480706691741943, -0.2980445623397827, 0.6227272152900696, -0.38722601532936096, 0.5213591456413269, 0.055103983730077744, -0.05485105514526367, 1.3739895820617676, 0.8113203048706055, -1.5398201942443848, 1.6013813018798828, 1.7374298572540283, 0.4206610321998596, -0.8983980417251587, 0.9812601804733276, -0.23233604431152344, 0.10826665908098221, 0.25448545813560486, 0.21426762640476227, -1.410695195198059, -0.7798459529876709, -0.21569395065307617, 1.0833443403244019, -0.05135347321629524, 0.32800987362861633, 0.2905471920967102, 0.6121284365653992, 0.5454583764076233, -0.054345015436410904, -1.6181119680404663, -1.6471613645553589, -1.639347791671753, 0.4299962818622589, -0.4004627466201782, -0.4967242181301117, 0.7496278882026672, 2.4697799682617188, 1.0632781982421875, 0.3487832844257355, 0.7924407720565796, 1.6350501775741577, 0.6558563113212585, -0.11157727241516113, -1.0500926971435547, -0.478441447019577, -0.8094555735588074, -2.2384142875671387, -0.7482247948646545, -0.9525847434997559, -0.2033218890428543], [0.024872133508324623, 2.239856004714966, -0.4021437168121338, 1.7772566080093384, 0.5838660597801208, 0.8164334893226624, 1.849027395248413, 1.3561946153640747, -0.17402249574661255, -0.4464717507362366, -0.7309726476669312, 0.6488574147224426, -1.2865031957626343, -1.5186042785644531, -0.16916970908641815, -0.16556021571159363, 0.5693149566650391, 0.9370669722557068, 1.012992262840271, 2.8599326610565186, 0.08171918243169785, 0.5859660506248474, -0.8419895172119141, 1.8090670108795166, 0.18531189858913422, 1.649987816810608, 0.8510854840278625, -1.0842633247375488, 0.035719048231840134, 0.9070826172828674, 1.9184486865997314, 0.06096420809626579, -1.2839770317077637, 0.11859966814517975, 1.062938928604126, -0.16396139562129974, 1.4289093017578125, 1.4817326068878174, 0.8887848258018494, -1.3738385438919067, 0.8764440417289734, -0.252305269241333, -2.903371810913086, 0.00018078032007906586, 0.7970970273017883, -0.04212401062250137, 1.1220042705535889, -2.0096116065979004, -0.8985037207603455, -0.9550327062606812, -1.1559407711029053, 2.5305564403533936, -1.065230131149292, 1.488195538520813, -1.1804958581924438, 0.7570679187774658, -0.08461862057447433, 2.027092695236206, 0.5860982537269592, 2.57258677482605, -0.28482043743133545, 0.02378252148628235, 2.499666452407837, 1.1153205633163452], [2.211881399154663, -0.4063473343849182, -0.3882887363433838, 2.016812324523926, 0.7089775204658508, 0.33988112211227417, -0.4770359694957733, -0.15507888793945312, -0.4388139545917511, -0.9927185773849487, 0.22605283558368683, 0.8216320276260376, 1.0531296730041504, 0.5521709322929382, 0.06881801784038544, -0.1314779669046402, 0.7560476064682007, -0.13924965262413025, 1.3631693124771118, -0.03797043114900589, -0.7435562014579773, 0.16454097628593445, 1.0363900661468506, 0.3663274049758911, -1.2522661685943604, 0.4601316750049591, -0.3251294195652008, 0.09824297577142715, -1.221531629562378, -2.0517520904541016, 0.24554021656513214, 0.037483420222997665, -1.4494659900665283, -0.06018538028001785, 0.7829110026359558, 0.4752705991268158, 0.0831747055053711, -0.2689829170703888, -0.06245610862970352, 1.2753419876098633, -2.223367214202881, -1.0179632902145386, -0.1263866126537323, -0.27624472975730896, -0.28481170535087585, -0.7574217915534973, 1.645514726638794, 0.2925150990486145, -0.669626772403717, 1.2324672937393188, -1.6879369020462036, 0.013773390091955662, 0.6719191074371338, 1.1596297025680542, 0.22237224876880646, 0.008904341608285904, -0.8987409472465515, 0.8048049211502075, 1.4068176746368408, -0.005460570566356182, 1.5776424407958984, 0.4384407103061676, 0.47600704431533813, -0.716852605342865], [-0.5113903284072876, 0.49078452587127686, -1.37493097782135, 0.11999012529850006, -0.8922358155250549, 0.5438456535339355, 0.22556646168231964, 0.3465864062309265, 0.4247962534427643, 2.0478971004486084, -1.2879139184951782, 0.6694046854972839, 0.20921486616134644, 0.3800303637981415, 0.14938382804393768, 0.4534095525741577, 0.6180819272994995, 0.32049375772476196, 0.536372184753418, 0.026180163025856018, 0.10628185421228409, -2.7976632118225098, 0.43568843603134155, 1.013474941253662, 1.3704309463500977, 2.529782772064209, -0.31030604243278503, 1.712027907371521, -0.22281625866889954, 1.249528408050537, -1.4442423582077026, -0.4319603443145752, -0.8950462341308594, 0.38888171315193176, 0.0916171744465828, 0.6758272647857666, -2.0049407482147217, 1.505411148071289, -0.8224878311157227, 0.7234723567962646, 0.32507508993148804, -1.6205805540084839, -1.1223998069763184, 0.37125909328460693, -0.9721566438674927, 0.23405708372592926, 1.0905545949935913, 0.994468629360199, 0.6228450536727905, 0.6033671498298645, -2.4688074588775635, 0.8739718198776245, 0.4123896360397339, 1.5818352699279785, -1.4614921808242798, 0.2515465319156647, 0.16672983765602112, 0.1465902030467987, 1.5736767053604126, 1.0351449251174927, 0.6843129396438599, -0.6081520915031433, 0.3069026470184326, -1.0061349868774414], [-1.4447824954986572, 0.7033443450927734, 1.5626755952835083, -0.3022352457046509, -0.1650122106075287, 0.6322342753410339, 1.6739333868026733, -0.02028302103281021, -0.5162420868873596, -1.4279004335403442, -0.299238383769989, 1.3108325004577637, -0.29454222321510315, 0.95998615026474, -1.3810486793518066, -1.6343698501586914, -0.22879669070243835, 0.4306464195251465, 1.4804760217666626, 0.007738584652543068, -0.9572122693061829, -0.40180256962776184, 0.3807116746902466, -1.4224231243133545, 0.48220938444137573, -0.1574966460466385, 0.7952176332473755, 0.794701874256134, 0.7991992831230164, -0.7625656723976135, -0.6206793189048767, 0.10253911465406418, 0.32397228479385376, -0.9196767807006836, -0.37980538606643677, -0.1974094659090042, 0.07757992297410965, -1.013810396194458, -0.7262937426567078, 1.241051197052002, -1.0436044931411743, 0.9639902114868164, -0.1564224660396576, 0.9064342379570007, -0.031160853803157806, 0.0633024349808693, 0.8594557642936707, 1.61154305934906, -0.3622802495956421, 2.6112112998962402, 1.6623326539993286, 0.06236949935555458, -2.1467764377593994, 0.5219113826751709, 1.189200758934021, -1.4700591564178467, 0.5925906300544739, 0.7328028082847595, 0.5683050751686096, 0.44362393021583557, -1.1539890766143799, 0.5167896151542664, 0.27839285135269165, -1.7858736515045166], [-0.3119576573371887, -0.5506428480148315, -0.25746509432792664, -1.5834252834320068, -0.20487990975379944, -1.5528686046600342, -2.600661039352417, 0.012447734363377094, 1.2415440082550049, 0.03192242234945297, 1.4560859203338623, -1.0200634002685547, 1.099642038345337, 0.4473036229610443, 0.047900956124067307, -0.2687511444091797, 0.8351670503616333, 1.875666618347168, -1.1614900827407837, 0.7701984643936157, 0.8060157895088196, -0.45864495635032654, 1.0098589658737183, 0.4970560371875763, 0.7648541927337646, -1.759533405303955, -0.5864958167076111, 0.6183972954750061, -0.39120006561279297, 1.1877840757369995, 1.5548090934753418, 0.07871323078870773, -0.13117927312850952, -0.29282423853874207, 0.7769613265991211, -1.041187047958374, -0.007270451635122299, -1.5505982637405396, 1.00113844871521, -0.9694576263427734, 1.081750750541687, -1.0655484199523926, -0.855771541595459, -0.65781569480896, -0.9366276264190674, -0.049470312893390656, -0.7579341530799866, -1.399122714996338, -0.6918571591377258, -0.3167233169078827, 0.8534337282180786, 0.692475438117981, 0.3545611798763275, 0.2100258469581604, -0.46948713064193726, -0.2241654247045517, 0.8861612677574158, 0.13399749994277954, 0.4860724210739136, -0.6504651308059692, -1.621557593345642, 0.668989896774292, -0.3623150587081909, -0.31394633650779724], [-0.34858784079551697, 0.09550491720438004, 0.9046915769577026, 0.01576821692287922, -0.24816083908081055, -0.2973535358905792, -2.0747084617614746, 0.696672797203064, 2.1645641326904297, -0.6678482294082642, 2.665205955505371, 1.6443793773651123, 0.37021148204803467, 0.3374978005886078, -0.4929603040218353, 0.2999075949192047, 0.8806672096252441, 0.07771140336990356, -0.548431396484375, -0.29592621326446533, -0.9232370257377625, 0.7841833829879761, 0.9980716109275818, -0.8259821534156799, -0.315967321395874, 0.09549453109502792, -1.420669674873352, 0.9971585869789124, -0.04874397814273834, 0.5662195086479187, -1.4172755479812622, -0.23757760226726532, 1.194424033164978, 0.3829309642314911, 0.21290212869644165, -0.906752347946167, 0.20459988713264465, -0.40809839963912964, 2.2479774951934814, -0.6348935961723328, -0.0992296114563942, -1.8624013662338257, -0.45368146896362305, 0.4800267517566681, 0.30895546078681946, 0.5600221753120422, -0.36309313774108887, 1.1189689636230469, 0.8347931504249573, -0.4113631248474121, -0.12315822392702103, -0.43007561564445496, -0.38083750009536743, 0.22623617947101593, 0.5927902460098267, 0.49597692489624023, 0.8460066914558411, 2.0147132873535156, -1.0327708721160889, 0.5276902914047241, 0.3048853874206543, -2.1076548099517822, -1.596392273902893, -0.4799268841743469], [1.104298710823059, -0.356724351644516, -0.6335887312889099, 0.036262013018131256, 0.8385008573532104, -1.2011346817016602, -0.6734592914581299, 0.8993422389030457, 0.7727454304695129, 0.5732774138450623, -1.0454537868499756, -0.7804323434829712, -0.09564325958490372, -1.1126954555511475, -0.5707900524139404, 0.23652391135692596, -0.5254127383232117, 2.5853993892669678, -0.3916066288948059, -1.335341215133667, 0.5946579575538635, -1.5102602243423462, 1.64426589012146, -0.17066100239753723, 0.6695809960365295, 0.5285061001777649, -0.282939076423645, 0.6558639407157898, -1.625938057899475, -1.7308266162872314, 0.268060564994812, 0.207771897315979, 2.4135422706604004, -1.8988803625106812, -0.09899517893791199, -0.046976491808891296, -0.3581232726573944, -3.2254786491394043, -0.6854640245437622, 0.1452929526567459, -0.20005646347999573, 2.155644416809082, 0.9472377300262451, -1.3743841648101807, 0.23524680733680725, -1.4352421760559082, -0.5276714563369751, 0.2840936481952667, -0.702545166015625, -0.14763890206813812, -1.123367190361023, -0.5640087127685547, -0.5158500671386719, -1.061307430267334, -0.47129952907562256, 1.0191209316253662, 1.4683656692504883, -0.9346098303794861, 1.2900618314743042, -0.07098262012004852, 0.020931385457515717, -1.1857354640960693, 1.5365393161773682, -0.4838206171989441], [-0.07694276422262192, 1.8988256454467773, 1.3294000625610352, -1.2649420499801636, -0.6612472534179688, -0.6813411116600037, -0.07956735044717789, 0.485364705324173, 0.08777013421058655, 1.326675295829773, 0.9893448352813721, -0.07217276841402054, -0.004464369732886553, -0.4092566967010498, -1.8525595664978027, 1.0323172807693481, 1.0047763586044312, -1.9849234819412231, -0.9227136373519897, 0.5677347779273987, 0.4719891846179962, 0.9926613569259644, -2.7155771255493164, -0.08064932376146317, 0.3873532712459564, 0.5129826068878174, -1.2299567461013794, 0.5443320870399475, -0.15452012419700623, 0.01850976049900055, 0.13810285925865173, 1.40939462184906, -0.7987154722213745, -0.9090450406074524, -0.32021498680114746, 0.17931178212165833, -1.144350290298462, -0.3632267415523529, -0.159234881401062, 1.877639889717102, 0.6771473288536072, 2.1182329654693604, 0.22188133001327515, 0.6833962798118591, 0.09874226897954941, 0.4545857310295105, -1.4797836542129517, -0.6799288988113403, -0.02946566604077816, -0.7099409103393555, 0.2618354558944702, 1.908981442451477, 0.5406326651573181, -0.6833752393722534, -0.7575660347938538, 1.1105492115020752, -1.4434155225753784, -1.817503809928894, -1.2147773504257202, 1.8077378273010254, 1.5982961654663086, 0.10060343891382217, 0.25254151225090027, -0.8396726846694946], [-0.6059549450874329, -1.0634660720825195, 0.7881917953491211, -1.5272631645202637, -0.6222953796386719, 0.9835540652275085, 1.152681589126587, 0.822236955165863, 0.2786402404308319, 0.11505718529224396, -1.5074000358581543, 1.1897804737091064, 1.586629867553711, -1.3898918628692627, -0.03326481953263283, 0.3453923761844635, -0.41211721301078796, -1.6008265018463135, 0.33600959181785583, -1.5377792119979858, -1.2293612957000732, -0.41089680790901184, -0.9444265961647034, 0.04949527233839035, 1.0601412057876587, 1.5005921125411987, -0.29761838912963867, -0.7234183549880981, 0.9236422777175903, 0.036233026534318924, 0.6151903867721558, -1.0017181634902954, 0.12800633907318115, 0.6138707399368286, 0.8901400566101074, 1.0520464181900024, 0.7853730916976929, 0.3851550221443176, 0.1073199063539505, -0.92789226770401, 0.43971529603004456, 0.7280213832855225, -0.5439230799674988, -0.3692886531352997, -2.2002174854278564, 0.5991092324256897, -1.4731537103652954, -0.24674749374389648, 0.006436101160943508, -0.727371335029602, -0.5278769135475159, -0.8266481757164001, -1.81206476688385, 0.18032145500183105, 0.7753077745437622, -0.8924121856689453, 0.22085189819335938, 1.5513757467269897, -1.1833043098449707, -1.7397143840789795, 0.9023537039756775, -0.9433687329292297, 1.200826644897461, 0.4192714989185333], [-0.029066335409879684, -0.8479418158531189, -1.9945820569992065, 0.3901735544204712, -2.747303009033203, 0.023342646658420563, -0.19478559494018555, -0.046001650393009186, 0.36847323179244995, -0.13038715720176697, 0.5872008204460144, -0.7196944355964661, 0.2287992537021637, 0.5319129824638367, 1.2136690616607666, -1.7180825471878052, 1.168709635734558, -0.8463063836097717, -1.764701008796692, 0.20131051540374756, 0.6246442794799805, -0.015613229013979435, -1.6794970035552979, -0.6594356298446655, -0.6401455402374268, -0.8964779376983643, 0.40369871258735657, -0.10969875752925873, 0.1841534823179245, -0.9753303527832031, -0.23385556042194366, -0.5121268630027771, 0.5293617844581604, -0.45681124925613403, -0.8104987740516663, 0.7189521789550781, -1.7058157920837402, -1.2343525886535645, -0.6422502994537354, -0.33001741766929626, 0.5514531135559082, 1.9220613241195679, -0.780552327632904, 2.19224214553833, -0.3176012635231018, -0.8501399159431458, 0.29741236567497253, -0.44239985942840576, -2.5696260929107666, -1.2908066511154175, 0.8500859141349792, 1.03103506565094, 2.342170000076294, 0.5102884769439697, -0.869634747505188, -1.442280650138855, -1.3785291910171509, 0.9312995076179504, 0.6551998257637024, 0.40720459818840027, -1.7518075704574585, -0.4762265086174011, -0.06386756151914597, -0.8246473073959351], [0.8085524439811707, 0.3881838917732239, -0.0186587143689394, -0.6228896379470825, 1.4067530632019043, -1.061300277709961, 0.5890733599662781, 0.2960703670978546, 0.5736696124076843, -1.206568717956543, -0.1752540022134781, 0.48170605301856995, 0.37229812145233154, -0.004763427656143904, 1.2711036205291748, -1.019860029220581, 0.7873144149780273, 0.35297536849975586, -0.7054247260093689, -0.6911208033561707, 0.44045349955558777, -0.5299023389816284, 0.0330798365175724, 0.8298152089118958, -2.0798890590667725, 0.2588214576244354, 2.5816237926483154, -0.4076545834541321, -0.7124325633049011, -0.11056584864854813, 1.6999777555465698, -0.4953616261482239, 0.8239635229110718, 1.7340847253799438, 0.014684466645121574, 0.7412866353988647, 0.025035211816430092, 0.20870114862918854, 0.39134833216667175, 0.3914793133735657, -0.23214271664619446, -3.1950762271881104, -0.9432740807533264, 0.6680604815483093, 0.5402810573577881, -0.478502482175827, 1.4298604726791382, -0.9004641175270081, 1.0374433994293213, -1.0871516466140747, -0.33349889516830444, 1.0524332523345947, 0.06307636201381683, 0.3186268210411072, -1.4222135543823242, 1.925682544708252, -0.6544692516326904, -0.43810200691223145, 0.932348370552063, 1.3049238920211792, 0.4310095012187958, -1.4749224185943604, -0.032823774963617325, 1.3767393827438354], [1.196563720703125, -0.5638108253479004, 0.638050377368927, 0.040264792740345, 0.647083044052124, 0.21182605624198914, 0.04662192985415459, 0.9436610341072083, 0.9619098901748657, 0.9778484106063843, 1.5302685499191284, -2.1718642711639404, 0.9399729371070862, -0.4181993007659912, -0.13683082163333893, 0.9356498718261719, 0.402630090713501, -0.8054103851318359, -1.7539782524108887, 1.2000279426574707, -0.6899879574775696, 1.226431965827942, 1.0252087116241455, -0.319877028465271, 0.38691118359565735, 0.7346146702766418, -0.5746517777442932, 0.23750223219394684, 0.07474395632743835, -0.30422887206077576, 0.022059375420212746, -0.5514228940010071, -0.8909136056900024, -0.965815007686615, 0.4246140718460083, 0.03552534803748131, 1.1017696857452393, 0.8996540307998657, 1.8587216138839722, 0.798751175403595, 0.10676635056734085, -0.3108300566673279, -0.6347301602363586, 0.44198307394981384, 0.6950689554214478, -0.3483392000198364, -0.9719869494438171, -0.062037717550992966, 0.6889245510101318, 2.205082654953003, 0.12351828813552856, -0.9595500826835632, 0.10328519344329834, -0.938580334186554, -0.37781766057014465, 0.808385968208313, 1.912291169166565, -0.21567751467227936, 0.5981179475784302, 1.2507712841033936, -1.2448976039886475, -0.5591328144073486, -0.8527349829673767, -0.7649700045585632], [0.9841529130935669, 0.5577026605606079, -0.569232702255249, 0.4786105155944824, -0.28905564546585083, 1.7150659561157227, 0.5341489315032959, 0.4260544180870056, -0.22307495772838593, -1.5673257112503052, -1.1024434566497803, 0.132010817527771, -0.05018901824951172, 1.318359375, 1.4811803102493286, -0.5577464699745178, -0.08143686503171921, -0.6855612397193909, 0.46659931540489197, 0.33359721302986145, 0.8995648622512817, 0.09739591926336288, 0.7830842733383179, -2.378000497817993, 0.15445193648338318, 0.7476674914360046, -0.8421790599822998, -0.49016183614730835, -2.793039083480835, 0.3334815800189972, -0.2894972264766693, -0.45005619525909424, 1.6334357261657715, -0.7172420620918274, 0.08926640450954437, 0.30825182795524597, -0.955342710018158, 0.766758918762207, -0.012787227518856525, 0.6253748536109924, -1.539717435836792, 0.5806682109832764, 0.7668299078941345, -1.5397974252700806, -0.5133579969406128, -0.40859776735305786, -0.2867361307144165, 0.4299522340297699, -0.24635225534439087, 1.34586501121521, 0.34859412908554077, 0.8745065331459045, -0.6377950310707092, 0.8353424072265625, -0.7470306158065796, 1.7973670959472656, -2.132262706756592, -0.6733250617980957, 0.3121800124645233, -0.18977589905261993, 1.4884511232376099, 0.4571543037891388, -0.4941480755805969, 1.0185045003890991], [0.1084892526268959, -0.8327293395996094, -0.08524394035339355, 0.265516072511673, -0.22515642642974854, -0.2930455207824707, -0.3612861931324005, 1.0169506072998047, -0.5707434415817261, 0.6487061381340027, -0.4207246005535126, -0.7222832441329956, -2.184113025665283, -1.5026824474334717, 0.7028902173042297, 0.1018269807100296, 0.3736586570739746, -0.21946464478969574, -0.7942919135093689, 0.7307658195495605, 0.9248098731040955, -0.4585227072238922, -0.584869921207428, -0.45927539467811584, -1.1721727848052979, -0.5497007966041565, -0.37259194254875183, 0.023373417556285858, -0.19369523227214813, -0.8756551146507263, -0.9352115988731384, 0.8107677102088928, -0.3743760585784912, -1.13486909866333, 1.4694883823394775, -1.0053294897079468, -0.8147070407867432, -1.901066541671753, -0.6985989212989807, -1.1133551597595215, -0.597620964050293, 0.3739771842956543, 0.9557885527610779, 0.8512946367263794, 1.6172457933425903, -0.04632078483700752, 1.116914987564087, 1.4989979267120361, -0.6042065024375916, 0.29867470264434814, -1.639195442199707, 0.5495120286941528, -0.23012246191501617, 0.5699114799499512, -0.46485811471939087, 0.7225075364112854, 0.782425045967102, 0.14370940625667572, 1.016656517982483, -0.040744077414274216, 0.516549289226532, -0.8190768957138062, -0.5981358289718628, 2.1169614791870117], [-1.23094642162323, -0.9459272623062134, -0.002946089021861553, -0.7583739757537842, -0.8119750022888184, -0.0026885047554969788, -0.22209197282791138, 0.7418854236602783, 1.399869441986084, 1.4762729406356812, 0.9649248123168945, 0.21872101724147797, -0.04672304540872574, 0.12621787190437317, -0.9144319295883179, -0.6238208413124084, 1.5589808225631714, -0.5144330859184265, 0.9044087529182434, 0.6950200796127319, -0.06782353669404984, 0.19338633120059967, -0.4059363305568695, 0.5477268099784851, -0.294428288936615, -1.7153388261795044, 0.2119351178407669, -0.7103487253189087, -0.4472736716270447, 0.790350615978241, -0.15090304613113403, 0.657854437828064, 0.9808515906333923, -0.4263627827167511, 0.01479710265994072, -1.1718686819076538, 0.05538787320256233, -1.2463864088058472, -0.8096529841423035, 1.5736860036849976, 0.6417968273162842, -0.9857960939407349, -0.8316925168037415, -0.6693857312202454, 1.4217467308044434, -0.5849601030349731, 1.4473602771759033, 0.8391783833503723, -1.503434181213379, 0.8908882141113281, 0.04627504572272301, 0.5997444987297058, -0.1232662945985794, -0.7174223065376282, -0.5572436451911926, 0.5046162605285645, -0.19477452337741852, -0.6512041687965393, 0.2748967111110687, 0.9014837741851807, -1.9079539775848389, -0.515098512172699, 1.446062445640564, -0.747326672077179], [0.9564217329025269, -0.272672176361084, -0.03239274024963379, -0.21345655620098114, 0.10189864784479141, -1.6659798622131348, 0.9954723119735718, -0.08730942755937576, 0.3492112159729004, 1.4629215002059937, -0.5750863552093506, -0.5926553606987, 0.6024826765060425, 0.6620029807090759, -0.7392056584358215, -0.5665601491928101, 1.8195276260375977, 0.7048893570899963, 0.10548306256532669, -1.5198025703430176, -0.9855319857597351, 0.8564859628677368, 1.8263297080993652, 0.3336560130119324, -0.4741036593914032, 0.03989628329873085, -1.0181376934051514, 1.0528641939163208, 1.1119120121002197, -0.4665273129940033, -0.5838784575462341, 2.0568292140960693, -0.43683722615242004, -0.26556870341300964, 1.1561347246170044, 1.0805391073226929, -0.08390256762504578, -0.9195430874824524, -1.4139113426208496, 1.3540055751800537, -0.12751248478889465, -0.784197986125946, -1.0033574104309082, -0.3228689730167389, 0.30583101511001587, -0.4212886095046997, -0.4023587107658386, -0.6776285767555237, -1.0601162910461426, 1.6318925619125366, 0.897815465927124, 0.8423488140106201, -1.6783630847930908, 0.3916487693786621, 1.5391740798950195, 0.025864411145448685, -1.1630319356918335, 0.32411515712738037, -1.08597993850708, -0.51138836145401, 0.8516446948051453, 1.106050968170166, 0.6203300356864929, -0.8704675436019897], [-0.1019340306520462, 0.8307344317436218, 1.6156240701675415, -0.24298761785030365, -1.0263389348983765, 2.0978097915649414, 0.18712255358695984, -0.852963387966156, -0.3522062599658966, 0.1307086944580078, -2.517418622970581, 1.223551869392395, -2.0408546924591064, 0.3265596628189087, -0.2286820113658905, -3.0544493198394775, -0.4103338122367859, 0.7412866353988647, 0.9677406549453735, 0.28922122716903687, 0.20072317123413086, 1.2528316974639893, -0.5112575888633728, 0.4189717769622803, 0.2648380398750305, -0.381523072719574, -1.7267463207244873, -0.8469547629356384, 0.1305263489484787, 0.7533137202262878, -0.25633683800697327, -0.08970451354980469, 1.0220088958740234, 1.1068114042282104, -0.2520195245742798, 1.6438146829605103, -0.14404608309268951, 2.362102746963501, 0.11565178632736206, -0.12830036878585815, -1.2389634847640991, -0.7584447860717773, 0.060934245586395264, 0.016951803117990494, 0.4003170132637024, -0.6478689908981323, 1.2649624347686768, 0.518268883228302, -1.128087043762207, -1.4740315675735474, 0.15944506227970123, -1.4395145177841187, -1.5229928493499756, 0.2721810042858124, -1.884670615196228, 0.214456245303154, -1.3242794275283813, 0.04441377893090248, -0.945484459400177, -0.3422907590866089, -0.5219156742095947, -0.5569051504135132, -0.8175132274627686, -0.22490400075912476], [0.5907139778137207, -0.08373591303825378, -0.2860468029975891, 1.3705003261566162, -1.3015645742416382, 1.325752854347229, 1.3820245265960693, 0.7815054059028625, 0.4315721392631531, 0.8069342374801636, -2.051372528076172, -0.22714436054229736, 0.037661682814359665, -0.8165819048881531, 0.8914048075675964, 1.0579386949539185, 1.0337074995040894, -1.0013192892074585, 0.4917616844177246, 1.0615872144699097, 0.5131996273994446, 0.017691172659397125, 0.5013178586959839, -0.05588434636592865, 0.601897120475769, -1.4484293460845947, 0.15950186550617218, 0.9381818175315857, 0.33898937702178955, 0.4138946235179901, 0.63759845495224, 0.3460288643836975, 2.2038469314575195, -0.4261334538459778, 0.9261466264724731, 1.2149572372436523, -0.05377177894115448, 0.7979685664176941, -1.0971938371658325, -0.5932886600494385, 0.06426052749156952, 1.8638612031936646, 0.7740233540534973, 0.4929754137992859, 0.5432029366493225, 1.6162232160568237, 0.8461911082267761, -0.7396464943885803, 1.1392630338668823, -1.3814200162887573, -1.3130056858062744, -1.4278901815414429, -1.3710657358169556, -1.784078598022461, 0.4763326942920685, -0.2583538591861725, 2.500231981277466, -0.03935079649090767, -0.36944881081581116, -1.7734395265579224, 0.280356764793396, -1.2782553434371948, -1.2714529037475586, 0.4921610653400421], [0.5493025779724121, -0.14972436428070068, -0.2574610710144043, -0.6013249158859253, 0.5404438972473145, -0.3239763677120209, 0.4640502333641052, -0.2915551960468292, -0.2802434265613556, 1.2115402221679688, 0.21772165596485138, -1.5549286603927612, -1.254112720489502, -2.2007458209991455, 1.13681960105896, 0.48716872930526733, 0.250074565410614, -1.0728976726531982, 0.7534357905387878, -1.0125536918640137, -2.0247421264648438, -0.06100080534815788, 0.3266027867794037, -0.3702199161052704, 1.1737009286880493, -0.45622286200523376, 0.34507519006729126, 1.2616318464279175, 0.7030436992645264, -1.2121776342391968, 0.51691073179245, -1.118156909942627, -0.5388772487640381, -0.41960349678993225, -0.9728938341140747, 0.47137007117271423, -1.045553207397461, -1.1115506887435913, 0.9671305418014526, -0.8408936858177185, 0.049496252089738846, -0.5630056858062744, 0.8389931917190552, 0.7941111326217651, 1.253373622894287, -2.1791136264801025, -0.9879170656204224, -0.586681604385376, -1.3822021484375, -1.1019291877746582, 1.713276743888855, -1.5967917442321777, -0.6355782747268677, -0.5139021277427673, -1.0662444829940796, -0.6042613387107849, 0.8438758254051208, 0.5532553195953369, 1.6073508262634277, -0.982797384262085, 0.22754471004009247, -0.4365914762020111, 1.3145582675933838, 0.563057541847229], [1.6411699056625366, 0.6358166933059692, 1.2786246538162231, 0.0443841814994812, 0.1778108924627304, 0.5494518876075745, -0.4513314366340637, -2.0277645587921143, -0.9446457028388977, -1.5345884561538696, -0.07526066899299622, 1.7470369338989258, 0.5574744343757629, -1.2239443063735962, -0.3149296045303345, -0.014833072200417519, 2.9415597915649414, 0.2789020538330078, -0.43669605255126953, -1.2350205183029175, 0.1500423699617386, -0.744614839553833, -0.21694645285606384, -0.576154351234436, -0.34180039167404175, 0.5469942688941956, -0.09337043017148972, -0.5110902786254883, -0.2011609822511673, 0.3465835750102997, -0.7798718214035034, -0.12866295874118805, -2.2343215942382812, -0.23352324962615967, -0.6761899590492249, -0.24432769417762756, 1.4989221096038818, -0.04121985286474228, -1.2797248363494873, 0.09254051744937897, -0.1856568306684494, 1.2095930576324463, 0.6025643944740295, 1.3675079345703125, -1.0963555574417114, -0.7280997633934021, 2.0805420875549316, 0.3241448700428009, -0.04144873470067978, -0.6713457703590393, -0.4454081654548645, 0.2176007479429245, 1.2605072259902954, 0.0628509670495987, -0.12849238514900208, 1.3876056671142578, -0.028955576941370964, 2.144115686416626, -0.13376469910144806, 1.083858847618103, -1.813811182975769, 1.229462742805481, -0.18479673564434052, 1.2507892847061157], [1.6645798683166504, 1.038051962852478, -0.7956883311271667, 0.029465170577168465, 1.3375589847564697, 0.04534514993429184, -0.4879206717014313, 0.9092594981193542, 0.7643628120422363, -3.0820157527923584, 0.8525989651679993, -0.8662570118904114, -0.611552357673645, 0.6295458674430847, 0.9632163643836975, -3.0732719898223877, 2.460644245147705, -1.4114370346069336, -0.9824831485748291, -0.587598443031311, 1.847869873046875, 0.5869811773300171, 0.5354135036468506, 0.6225337982177734, 0.7629119753837585, -1.2910575866699219, 0.7114986777305603, -0.6832414269447327, -0.5752577781677246, 0.4446556270122528, -0.47233524918556213, 0.8386135697364807, 0.02615574188530445, 0.22847603261470795, -0.40050992369651794, 0.6623061895370483, -0.7524377703666687, 0.14666712284088135, 1.9538925886154175, -1.0668048858642578, -0.9717307686805725, -0.5258916020393372, 0.10207340866327286, -0.3799976408481598, 1.4132331609725952, 0.15162822604179382, -0.09782636165618896, 0.6092435717582703, 0.7018579244613647, -0.09554851800203323, 1.6384587287902832, -1.5578233003616333, -1.0801728963851929, -1.2260690927505493, -0.05512173846364021, 0.8766183853149414, 1.1866439580917358, 0.6016270518302917, 1.556368350982666, -2.0251779556274414, 1.4081605672836304, 0.1490412950515747, -0.0960737094283104, -0.32502037286758423], [-0.7300891280174255, -0.6772269606590271, 0.20473666489124298, 1.0754014253616333, 1.9552319049835205, 1.4780012369155884, -0.11662072688341141, 0.19421853125095367, 0.1283116638660431, -0.4769986569881439, 1.1383373737335205, 0.08364947140216827, -0.7046542763710022, 1.6553707122802734, 2.0800538063049316, 0.5095487236976624, 1.7552731037139893, 0.8091944456100464, -1.438185214996338, 0.556596040725708, 3.0817620754241943, -0.06161881238222122, -2.4750149250030518, -0.8931957483291626, -0.7150745987892151, 0.17191757261753082, -0.49224022030830383, -0.40235915780067444, 1.2831230163574219, -0.4609531760215759, 1.9837090969085693, 0.06340517848730087, 2.327816963195801, 2.5532593727111816, -0.05957568436861038, -0.8802866339683533, -0.19739502668380737, -0.7283003926277161, -3.2348551750183105, -1.7729281187057495, 0.18530991673469543, 0.44504469633102417, 0.4182657301425934, -0.32285842299461365, 0.839460015296936, 0.1459040641784668, -0.22369223833084106, -1.6832079887390137, 0.5467193722724915, 1.2734484672546387, 0.18996642529964447, 1.6401718854904175, -0.12931233644485474, -0.390044242143631, 0.9927502870559692, -1.3563355207443237, 0.8484724760055542, -0.9967965483665466, 0.9315215349197388, -1.1582046747207642, -0.6718868017196655, -1.4735863208770752, 1.3981010913848877, -0.898612916469574], [-1.88247811794281, 0.6678163409233093, -0.42544013261795044, -0.5962387919425964, -0.0980222076177597, -0.03253836929798126, 0.09577858448028564, 1.9581936597824097, -1.846369981765747, -1.074643611907959, 1.0255407094955444, -0.41536906361579895, -0.7330794930458069, 0.7464724183082581, -0.4466947615146637, -0.02095789648592472, 1.4491711854934692, -0.347881942987442, 0.24175383150577545, -0.8262366056442261, -0.7807548642158508, 1.5986286401748657, -0.4499265253543854, 0.9200740456581116, -0.6278311014175415, -1.3716315031051636, -0.27580249309539795, -0.002036250662058592, -0.35638710856437683, 1.335845947265625, 0.37944725155830383, -1.150380253791809, -0.649919867515564, 0.3291868567466736, 0.8028379082679749, 0.9684171080589294, 0.013658320531249046, 0.009298797696828842, -0.24035368859767914, 0.8468872904777527, 1.4466838836669922, 0.33335620164871216, 1.4199855327606201, 0.940597414970398, 0.9999895095825195, 1.6864206790924072, 0.18718776106834412, 0.5502689480781555, -0.6516930460929871, -0.7571895122528076, 0.795195460319519, 0.26527199149131775, 0.4944302439689636, 0.7394063472747803, 1.1926813125610352, 0.23741842806339264, 1.7375344038009644, -0.39431750774383545, 0.2122495174407959, 0.3262627124786377, -0.5354980826377869, -1.179659366607666, 0.19730144739151, 0.4778153598308563], [1.151059865951538, 0.4527692496776581, 0.10045584291219711, 1.152773380279541, -2.8625481128692627, 0.8686764240264893, 0.13318488001823425, 1.0087167024612427, -0.19681987166404724, 1.6068533658981323, -1.3558813333511353, -0.2952735722064972, 0.823884904384613, 0.2953198552131653, 0.3434467911720276, 0.9950178265571594, 1.1343662738800049, -1.555895447731018, -1.0419623851776123, 1.3074136972427368, -0.7647807598114014, 0.12134931236505508, 0.6499860286712646, 0.7302407026290894, 1.4996970891952515, 1.4591726064682007, -0.4332432746887207, -0.4424152672290802, 1.7352627515792847, 0.4253137707710266, -1.1078383922576904, -0.038914527744054794, -1.57035493850708, -1.1366673707962036, -0.9738389253616333, -1.9360531568527222, 0.5750423073768616, -0.278201162815094, -0.2128557562828064, 0.3993265628814697, -1.229364037513733, -0.8463471531867981, -2.2754383087158203, 0.6478231549263, -0.15301772952079773, 0.7310075163841248, 1.2287856340408325, 2.0392658710479736, -0.08839207887649536, -0.10393627732992172, 0.7347641587257385, -1.1731404066085815, -0.7182488441467285, -0.1547606736421585, 1.2343180179595947, 0.4329751431941986, 0.8838437795639038, -0.8094483017921448, 0.6002349257469177, -0.23800590634346008, 0.10624178498983383, -0.37281903624534607, 1.6171756982803345, 0.17825953662395477], [0.05308578908443451, -0.09275489300489426, 0.3249366283416748, -2.133267402648926, -0.9570409655570984, -0.40339159965515137, 0.1395314782857895, 0.5127807259559631, -1.3527418375015259, -0.237189382314682, 0.24665023386478424, 1.5087943077087402, -2.097820997238159, -1.1052831411361694, -0.06279700249433517, -1.3952269554138184, 1.8306013345718384, -0.09622929245233536, 0.21201089024543762, -0.7341010570526123, -0.018062887713313103, -1.2458842992782593, -1.910355806350708, -0.3388136327266693, -0.04551880434155464, -0.38720497488975525, 1.8275554180145264, -0.03485862538218498, 0.21824684739112854, 0.2000025510787964, -0.41000568866729736, 1.4086241722106934, 0.5046241879463196, -0.40598905086517334, 1.2511813640594482, -0.7225990295410156, -0.04867381975054741, -0.6881620287895203, 1.2877485752105713, -0.4521103501319885, -0.7951824069023132, -0.5414422154426575, -0.45270416140556335, -0.9325507879257202, 0.913801908493042, -0.16985982656478882, 0.6811960339546204, -2.1867949962615967, 0.7387043833732605, -1.5248143672943115, -0.14223481714725494, 0.5312548875808716, -0.1488993614912033, 0.6340106725692749, -0.4276350140571594, 0.5684053897857666, 0.8932459354400635, -0.21707013249397278, 0.18572379648685455, 0.2409457266330719, -1.4784094095230103, 0.81252121925354, 0.4651385545730591, 0.6010962724685669], [2.8661611080169678, 0.8256785273551941, -2.6777896881103516, -1.23599374294281, -1.6609132289886475, -0.8074824810028076, -0.14647802710533142, 0.8447251915931702, -1.1801849603652954, -1.0927891731262207, 0.4942641258239746, -0.581311047077179, 0.46111443638801575, -0.1168687641620636, -1.8165456056594849, -0.827674925327301, 2.152012825012207, 0.4510197341442108, -0.17970407009124756, -0.712746262550354, -0.8911582827568054, -0.8146466612815857, 0.7724172472953796, 0.12442970275878906, -0.318233460187912, 0.6255196928977966, -0.7096537947654724, -0.4327540695667267, 0.4215949475765228, -0.45721203088760376, 1.792090892791748, -1.6971440315246582, -0.04089987277984619, -0.9736270308494568, 0.6661187410354614, -0.06118645891547203, 0.8022418022155762, 1.020909070968628, -0.22993244230747223, -2.0052521228790283, 0.7448137998580933, -0.7402285933494568, -0.6393601894378662, 0.2504312992095947, -1.6933252811431885, -0.36593908071517944, 1.1521763801574707, -0.17666271328926086, -1.2201982736587524, 1.3785728216171265, -1.433949589729309, 0.2681053876876831, -0.22934673726558685, 0.9249368906021118, -0.6575971245765686, 1.2052853107452393, -0.41079336404800415, -1.4469118118286133, 1.2094806432724, -2.370978832244873, -0.6076205372810364, -0.8343160152435303, 0.7967577576637268, 1.3631685972213745], [-1.0530450344085693, 2.121887683868408, 0.4538121521472931, 0.3755824565887451, 0.392927885055542, -2.090938091278076, -0.2976464331150055, 0.5792864561080933, 0.07851312309503555, 1.4495307207107544, 0.4964908957481384, 1.777001142501831, -0.2768109440803528, -0.29617583751678467, 0.1418948620557785, -1.695380449295044, 1.288046956062317, -0.24283170700073242, 0.6603875160217285, 0.3536651134490967, 1.9511675834655762, -0.023031918331980705, -1.5980955362319946, -0.9157567024230957, 0.13133065402507782, -0.9040283560752869, -0.5897586345672607, -1.7727657556533813, -0.44581499695777893, -0.12499231845140457, -0.08378338068723679, -0.7897660732269287, 0.2577957808971405, -1.7426903247833252, -1.2768092155456543, -0.2194565087556839, 0.9529587626457214, 1.1322455406188965, 0.6746366620063782, 2.0454905033111572, -0.20908571779727936, 1.3288792371749878, 0.10862007737159729, -0.867367684841156, -0.21246019005775452, 0.2993258535861969, -1.817694902420044, 0.3679516017436981, 0.7693865895271301, -1.348406434059143, -0.06372427940368652, -0.009121382609009743, 1.181530237197876, 2.1858067512512207, 0.6109066009521484, -0.20400728285312653, -0.6819273233413696, -0.619591236114502, 0.8449639678001404, 0.8114868998527527, -1.1016261577606201, 0.03843943774700165, 0.6309204697608948, 0.1720791459083557], [-0.8116270303726196, 0.0880107581615448, 1.9429738521575928, -2.0547871589660645, -0.5885433554649353, 0.24725022912025452, -0.5846782922744751, 0.0985879898071289, -0.4664508104324341, -0.9357104301452637, -0.23401443660259247, 1.3482316732406616, 0.33060401678085327, 0.24100248515605927, 2.306199312210083, 0.9402550458908081, -1.4926213026046753, -0.578957736492157, 1.8657852411270142, -0.4820447266101837, 0.22593717277050018, -0.7583833932876587, -0.554611325263977, -1.6830166578292847, -1.7681339979171753, -1.1871819496154785, 0.10528327524662018, 2.5467214584350586, 0.16920672357082367, -0.3508546054363251, 0.9650647044181824, -0.5463487505912781, -0.39126935601234436, 0.562964141368866, -0.4870857894420624, 0.6219422221183777, -1.1631959676742554, -0.7897711396217346, -1.4658279418945312, -0.9336022734642029, -0.12182430177927017, -0.2976509630680084, -0.9926219582557678, 1.138334035873413, -0.5754224061965942, -0.18749822676181793, -1.530339002609253, -0.22437801957130432, 0.6952292919158936, -0.3029947578907013, 0.2904627025127411, 2.8305180072784424, -0.0274796225130558, 0.03248751536011696, 0.08050484955310822, -0.39788034558296204, -0.7274386286735535, -1.2292673587799072, 1.8199161291122437, 1.4553937911987305, 2.0905940532684326, 0.783970832824707, 0.15976089239120483, -1.0247102975845337], [-0.700714111328125, 0.5622242093086243, 1.4946290254592896, 0.17908941209316254, -0.9218160510063171, -0.9351518154144287, -0.4961335062980652, -0.09115726500749588, -0.007932918146252632, 0.8935789465904236, 0.005363216623663902, -1.4230257272720337, 0.41459307074546814, 0.276276558637619, 0.27881520986557007, 1.2979907989501953, -1.3570804595947266, 0.3112614154815674, 0.7328193187713623, 1.2545852661132812, 1.145810604095459, -0.34439918398857117, -1.2614707946777344, 1.4666492938995361, -0.1491333246231079, 1.554379940032959, -1.3652769327163696, 1.2462126016616821, -1.432093620300293, 0.5434487462043762, 0.45609211921691895, -1.1907342672348022, 0.8651296496391296, -0.3737366497516632, 0.1305919587612152, 0.33458423614501953, 1.542092204093933, 0.9203668236732483, 0.44818398356437683, 1.4663432836532593, 2.9146316051483154, -0.4908139705657959, -1.436549425125122, 1.0224361419677734, 0.04354174807667732, -1.6513817310333252, -0.5310470461845398, -0.21397829055786133, 1.3976191282272339, 1.3433970212936401, 0.5621232986450195, 1.4718773365020752, -1.3702442646026611, 0.36037755012512207, 1.7932511568069458, 0.8486778140068054, 1.0079278945922852, 0.6328962445259094, 1.8510284423828125, 0.19139036536216736, 1.6072593927383423, 0.08981679379940033, -1.5178457498550415, 0.02213749848306179], [-0.7493635416030884, -0.09505435079336166, -1.0537837743759155, 0.007492630742490292, -1.7055997848510742, -0.12336075305938721, 0.911027729511261, 0.7741685509681702, 1.523354172706604, -1.1211934089660645, -0.1446736752986908, -0.7299696207046509, -0.15634243190288544, -0.7059614062309265, -0.6867485642433167, 0.4109684228897095, -0.9869093298912048, 0.5851680040359497, -1.0079354047775269, 0.7074259519577026, -1.430422067642212, -0.7135044932365417, 2.4001526832580566, -0.6103618144989014, -0.5280840992927551, -0.9055519104003906, -0.8370832204818726, 1.713580846786499, -1.6113007068634033, -0.8652076721191406, -0.10110048204660416, -1.6317118406295776, 0.20177844166755676, 0.5811863541603088, 0.7768401503562927, -0.01811865158379078, 1.026930809020996, -0.26325300335884094, -0.012242686003446579, 1.4205670356750488, 0.4254189431667328, -0.47506195306777954, -1.5271967649459839, -0.04306716471910477, -1.9653736352920532, -0.06601788103580475, 0.37384337186813354, 1.1273524761199951, 0.5106409788131714, -0.6006150245666504, -1.4325509071350098, -1.2690260410308838, 1.890031099319458, -0.19636276364326477, 0.28435519337654114, 0.007066749967634678, -1.312658667564392, -1.255266785621643, 0.10222801566123962, 0.2551640570163727, 0.5890687108039856, 0.281693696975708, -0.6479759216308594, -0.4961240887641907], [0.3549784719944, 0.028002293780446053, 0.9398778676986694, 0.25480255484580994, 0.6274744272232056, 0.1654263138771057, 0.012347787618637085, 1.6391193866729736, 0.6994104385375977, -0.31296253204345703, -0.15937888622283936, 0.5751022696495056, 0.5493292212486267, 1.0077162981033325, 0.6961987614631653, -1.3713200092315674, 0.22535184025764465, 0.31375929713249207, 1.1287587881088257, -3.0795505046844482, 0.30704614520072937, -0.5335612297058105, -0.007364178542047739, 0.0008648941875435412, -0.5413207411766052, -0.681389570236206, 2.1135058403015137, 1.2523423433303833, -0.2962341904640198, 0.4650938808917999, 0.4601532518863678, -1.1350266933441162, -0.5617097616195679, -1.1343111991882324, -0.9028543829917908, 0.008914231322705746, 0.10377220809459686, -0.6098653674125671, 0.28623726963996887, 0.2511821389198303, -0.3406231105327606, -0.46298256516456604, 0.6214037537574768, -0.05179256573319435, -2.1228320598602295, 0.19521787762641907, 0.25730833411216736, 1.1747119426727295, 2.4440860748291016, -0.3119942247867584, 1.867483377456665, -0.5586416721343994, 1.3785854578018188, -2.0847551822662354, 1.2133738994598389, 0.7164584398269653, 0.08557283878326416, -0.428854376077652, 0.18551109731197357, -0.8053920865058899, 1.7861510515213013, -0.7430951595306396, -0.9752386212348938, 0.31269845366477966], [-0.8088867664337158, -0.03336816653609276, 0.15798988938331604, 0.3116595447063446, 0.6014958620071411, 0.675376296043396, 0.7556746006011963, -1.3776094913482666, -0.5593327283859253, -1.1541926860809326, 0.29432645440101624, -0.6542677879333496, 1.3723633289337158, 0.48098286986351013, 0.5145821571350098, -1.4813368320465088, 1.2856935262680054, 0.8018226623535156, -0.4973483085632324, 1.402127742767334, 0.07717127352952957, 0.05093834176659584, 0.6048048734664917, 1.1345034837722778, 0.7310310006141663, -0.476685106754303, 1.591618299484253, -0.3465261161327362, -0.7891480326652527, -0.2754140794277191, -1.2237780094146729, 0.11004285514354706, 0.4888857901096344, -0.09370426088571548, -0.9030904173851013, 0.4848438799381256, -1.131181240081787, -0.8390201330184937, -0.4851647913455963, 1.6371831893920898, -0.18139879405498505, -2.400437593460083, 1.2050223350524902, -0.8327209949493408, 1.2897483110427856, -0.22049933671951294, -1.7705518007278442, 0.47282296419143677, 0.7167167067527771, 0.008603799156844616, 1.0872479677200317, 1.242419958114624, -0.8632110953330994, 0.5890636444091797, 0.7909336686134338, -0.28006744384765625, 0.730614960193634, 0.5929543972015381, -0.03544138744473457, 0.7130228281021118, 0.4628550708293915, 0.06125136464834213, -0.40002351999282837, 0.4933713972568512], [0.296634703874588, 0.1084020733833313, 0.009758785367012024, -0.5930942296981812, -0.5644107460975647, 0.7557507753372192, 0.5310704112052917, -1.9312891960144043, 0.7392975687980652, 1.2740830183029175, -2.377941370010376, 0.3035445809364319, 1.0410630702972412, 0.28524866700172424, -0.5495805144309998, -1.092626690864563, 1.6139479875564575, -0.047598306089639664, 0.5490857362747192, -1.126672625541687, -0.5366984605789185, -0.4285239279270172, 1.1873598098754883, 0.6047837138175964, -1.6473017930984497, -1.1234426498413086, -0.7601739764213562, -0.6626198291778564, 0.5817486047744751, 2.2320199012756348, 0.9797161221504211, 0.41471853852272034, -0.08826714754104614, 3.3190674781799316, 0.12393202632665634, -1.767279863357544, 1.5219401121139526, 0.13333268463611603, 0.5288683772087097, -0.03831535577774048, 0.6041482090950012, 0.09130758047103882, -1.273276448249817, 0.014090967364609241, 1.0018287897109985, -1.6457208395004272, 1.0774052143096924, 0.4535219967365265, -1.3231885433197021, -2.6948351860046387, -0.19892515242099762, -0.9999696612358093, -1.4326565265655518, 1.678137183189392, -0.26497992873191833, -0.323803186416626, -0.8630964159965515, -0.8774043917655945, -1.0683337450027466, -0.11360006034374237, 0.6567262411117554, 0.7018430829048157, 0.7380889654159546, -0.7078408598899841], [0.0427187904715538, -0.9301521182060242, 1.0127930641174316, 0.8503131866455078, 1.8356672525405884, -0.5191602110862732, 0.2636966407299042, -0.6636260747909546, -0.2417706549167633, -0.2803865075111389, 0.9125432968139648, -0.43650245666503906, -0.36341074109077454, -1.07071053981781, -0.4735777974128723, -1.6253750324249268, 0.765487551689148, -0.5786470174789429, -0.7756897211074829, 0.8532587885856628, -1.4191652536392212, -0.15001358091831207, -1.50395929813385, -0.3073728084564209, 0.3905649781227112, 0.05114402249455452, -0.3343554139137268, -0.015930453315377235, -0.6840398907661438, 1.12252676486969, -0.5925054550170898, -0.17790654301643372, -0.39667394757270813, 1.5563486814498901, -2.149143695831299, -0.175333634018898, 0.5885013937950134, -0.7854838371276855, 0.38267210125923157, 0.9322802424430847, 0.9434356093406677, 0.32162901759147644, 1.7316137552261353, -0.08368393033742905, -1.634615182876587, -1.8466148376464844, 1.0065946578979492, 0.15326657891273499, -0.14670412242412567, -0.8440958857536316, 0.32324647903442383, -0.42110130190849304, -0.8972918391227722, 0.19091801345348358, -0.062335047870874405, 0.45686522126197815, 1.0370420217514038, -0.8685598969459534, 1.149361491203308, -0.1301213651895523, -0.6937003135681152, 0.5657558441162109, -2.308380365371704, 0.05389046296477318], [-0.4339815676212311, -0.4474075436592102, 0.4242599308490753, 2.66536545753479, -0.6688620448112488, -0.49916377663612366, -2.6296863555908203, 0.4976666569709778, 0.07931110262870789, -0.4924144148826599, -0.16599448025226593, -0.8071560859680176, 0.13281556963920593, 1.3494970798492432, -0.9484167695045471, 0.7901173830032349, -0.125246062874794, -0.6968995332717896, -0.3699782192707062, 0.6172386407852173, -0.03512971103191376, -0.4607384502887726, 0.014453809708356857, -1.144170880317688, -0.8364355564117432, 0.4792557954788208, 0.37857744097709656, 0.35339146852493286, -0.876679539680481, -0.5476091504096985, -1.1832623481750488, -0.7472107410430908, -0.724287748336792, -0.9891965985298157, 1.012477159500122, -0.7660658955574036, -1.0873112678527832, -0.2897379398345947, 0.6618385314941406, 1.238317847251892, 0.5901420712471008, -0.6313713788986206, 1.8472274541854858, -1.1593950986862183, 1.2795714139938354, 0.839999794960022, -0.5783630013465881, -0.008464711718261242, 0.9852831959724426, 0.03283093497157097, 1.4503737688064575, -1.7563868761062622, -1.2174066305160522, -2.873896360397339, -0.6265835165977478, -0.32950514554977417, 0.1977405697107315, 2.09132981300354, 1.2499293088912964, 0.9602281451225281, -1.0974693298339844, -0.797322154045105, -0.18182674050331116, -0.2812088429927826], [1.2338818311691284, -0.3177391588687897, 0.9149080514907837, 2.084033727645874, 0.3111865520477295, -1.2641048431396484, -1.158855676651001, -0.2355707734823227, 0.9563959240913391, -0.32866302132606506, -0.6217131018638611, 2.099804401397705, 1.263883113861084, -0.17748881876468658, -0.2377043217420578, -2.1353440284729004, -0.4538986384868622, -2.0602400302886963, 0.5055128931999207, 0.1471778154373169, 0.32272830605506897, -2.5286643505096436, -1.1963919401168823, 1.0237730741500854, 1.291229486465454, -0.06835980713367462, 0.05359702929854393, -0.7535755634307861, -0.5672927498817444, -0.16619113087654114, -0.322615385055542, -0.5565450191497803, -2.5240912437438965, 0.9367349743843079, -0.7450979351997375, 1.0952874422073364, 0.6970727443695068, 0.6577650904655457, -0.5811553001403809, -0.7734003067016602, 0.0417969673871994, 0.017777632921934128, -0.41315141320228577, 0.5820567607879639, 0.6071659326553345, -0.10308579355478287, -0.4420236051082611, -0.16106517612934113, -0.5411705374717712, -2.049592971801758, -0.6758694052696228, 0.8347640633583069, 0.2066020667552948, 0.695073664188385, 1.0206496715545654, -0.15428759157657623, 2.271137237548828, -0.8972694277763367, -0.4611639082431793, 1.963476300239563, 0.4170202612876892, 2.12740421295166, 0.5410168170928955, 0.21773843467235565], [-2.260371446609497, -1.1595531702041626, 0.7910774946212769, -0.05674762278795242, 0.6624549031257629, 1.0822830200195312, -0.14379721879959106, 0.2514007091522217, 2.025423765182495, 1.2099443674087524, 0.07558245956897736, 0.3901212215423584, -0.093891941010952, 0.5373308062553406, -1.5392502546310425, 0.4587426483631134, 0.8995881080627441, 0.7065637111663818, 0.4915958344936371, -0.6970810890197754, -1.0776838064193726, -0.4097123444080353, -0.1077226772904396, 0.3803535997867584, 0.49966979026794434, 0.5843065977096558, -0.6537870764732361, 0.9152116775512695, -0.3837639391422272, 0.09190668165683746, 0.5148919820785522, 0.976815938949585, 1.0168061256408691, 1.0288920402526855, -0.7870703935623169, -0.22266365587711334, 0.2062181681394577, 1.3638278245925903, 0.24313168227672577, -1.0517842769622803, 1.2594947814941406, -0.07062473148107529, -0.5653942823410034, -0.7530588507652283, -0.7693097591400146, 2.754971504211426, -0.4079519808292389, 1.2821017503738403, -1.6565934419631958, 1.7251828908920288, -0.7656807899475098, 0.9156556129455566, -0.2776426672935486, 0.19371724128723145, -0.5829918384552002, 0.13786013424396515, -0.32551291584968567, 0.9897719025611877, -1.2387560606002808, -1.5935603380203247, -1.3633224964141846, 1.769707441329956, -0.48339056968688965, 1.3879222869873047], [1.4972416162490845, 0.431487500667572, -0.9290075898170471, 1.0819395780563354, -1.2072179317474365, -2.1448590755462646, -0.26694411039352417, -0.1720414012670517, 0.7426322102546692, 1.4947106838226318, 0.01034857053309679, -0.1352851837873459, 2.906667470932007, 0.7484940886497498, 0.03496494144201279, -0.6295979619026184, 0.5367969274520874, -0.28684523701667786, 0.04929732903838158, 0.544918417930603, 0.7795103788375854, 0.606968879699707, 1.1558005809783936, 1.905192494392395, 0.4245283603668213, 0.5740146040916443, -0.8413485884666443, -0.3408620059490204, 2.153413772583008, -0.2866124212741852, 0.41315075755119324, -2.168841600418091, -1.2254395484924316, 0.8899524807929993, -0.05815901234745979, -0.6031656265258789, 0.2916666567325592, 0.21980610489845276, -0.5189955234527588, -0.28440624475479126, 0.10146596282720566, -0.3314955234527588, 0.17454837262630463, -0.3387208878993988, -0.2293764352798462, -1.1712125539779663, 0.8500531911849976, -0.7711929082870483, 1.2860183715820312, -1.3098313808441162, 1.2296819686889648, 0.02402685396373272, -2.970848321914673, -0.7458195090293884, -0.6382681131362915, 0.642794668674469, -0.20141170918941498, 0.4507143199443817, 0.2406177520751953, 0.4460010528564453, 1.2102221250534058, 0.8708131909370422, -0.20432503521442413, -2.460909605026245], [-0.16427719593048096, -2.1073756217956543, 0.3536251187324524, 0.7820518612861633, 0.07032830268144608, -0.8727612495422363, -0.15718114376068115, -1.6182729005813599, -1.6920853853225708, 0.24944031238555908, 0.03650956228375435, 1.865684986114502, -0.7760869860649109, 1.3609575033187866, 0.06127122789621353, -0.18427090346813202, -0.05103301629424095, 0.01847047358751297, 1.272848129272461, 1.3181270360946655, -1.7602627277374268, 0.20019002258777618, -0.6835533380508423, -0.03810565546154976, 0.25316715240478516, -0.27430248260498047, 0.3725113868713379, 0.036501213908195496, -0.9668756127357483, -1.1568337678909302, -0.466621458530426, 1.4562007188796997, 1.3359328508377075, -0.5544063448905945, 0.9181379675865173, -1.0712416172027588, 0.30791687965393066, 1.500550389289856, -0.3361574709415436, 0.8007700443267822, 0.4378439784049988, -0.9531310200691223, -1.0134700536727905, 1.7738555669784546, -0.707030177116394, -0.028534071519970894, 0.5339399576187134, -0.585306704044342, 0.4573908746242523, -0.44705769419670105, -1.377221941947937, 1.1402138471603394, 0.5376072525978088, -0.4809514582157135, -1.0001167058944702, -0.8866086602210999, -1.3555541038513184, 0.4867471754550934, 0.7626450657844543, -0.1523311883211136, -0.46808508038520813, 0.7766544818878174, 0.05888815596699715, -0.6668558716773987], [1.3313699960708618, -0.6952401399612427, 1.6567002534866333, 0.30288997292518616, 0.8163678050041199, 0.9851766228675842, 0.6929270029067993, -0.10270064324140549, 1.373086929321289, -1.2926831245422363, -1.1617323160171509, -1.9030506610870361, -0.648862361907959, -0.37936991453170776, -1.3206250667572021, 0.545525074005127, 1.264864206314087, -0.9782522320747375, 0.27092471718788147, -0.16048946976661682, 0.10478886216878891, -0.9327443242073059, 0.3835640251636505, -0.2483622282743454, -0.04023829847574234, 1.9483976364135742, 0.20353257656097412, -0.13874764740467072, -0.10389285534620285, 0.22383737564086914, -0.3558003008365631, -0.1704772561788559, 0.7860923409461975, -1.0606472492218018, 1.285895824432373, -0.15015988051891327, -0.8068161606788635, -0.8563673496246338, 0.1863478422164917, 0.38614293932914734, -1.7477688789367676, -0.8447474837303162, 0.8133516311645508, 0.9184842109680176, 0.5177733302116394, -0.3325243592262268, 1.0611335039138794, 1.1587193012237549, 0.4122338593006134, 0.3486778438091278, -0.2468721866607666, 0.9359237551689148, 0.4435844123363495, 0.11207978427410126, -0.7101995944976807, 0.5266913771629333, 1.2993431091308594, 0.10903000831604004, -0.7060380578041077, 0.5272882580757141, -1.036104440689087, 2.4827401638031006, 0.5397325754165649, 1.3177772760391235], [-0.04980631172657013, -0.8707651495933533, -0.04412439465522766, 0.7367538809776306, 0.12885414063930511, 0.45398885011672974, -0.8143934607505798, 0.27025723457336426, 0.2091503143310547, 0.977921724319458, -1.2970603704452515, -0.2556142807006836, 0.2963903844356537, 0.9102939963340759, 0.5914616584777832, -0.31924062967300415, -0.10439079999923706, 0.0543711856007576, 1.0646209716796875, 0.7244191765785217, 0.350696325302124, 1.065028429031372, 2.011406421661377, -1.3244273662567139, 0.2846461534500122, 0.8125397562980652, 2.446341037750244, 0.5513784885406494, -0.875975489616394, -1.451320767402649, 0.932822048664093, -0.862943708896637, 1.4150562286376953, 1.4963886737823486, 0.3371905982494354, 0.8360967636108398, 0.6708974838256836, -0.34001684188842773, -0.1871524155139923, -0.35299986600875854, -0.7180490493774414, -0.5932114720344543, 0.1476241499185562, -0.5286975502967834, -0.15793515741825104, 0.7186341881752014, -0.6962472796440125, 1.0280098915100098, -0.5756271481513977, 0.10119805485010147, -0.24627919495105743, -1.5307871103286743, -1.0125612020492554, 0.020419957116246223, 1.2924221754074097, 0.1095358207821846, 0.3029087781906128, 0.5208311080932617, -1.290665626525879, 0.9048973917961121, -0.724242627620697, -0.13795745372772217, 0.4385521113872528, 0.0883953720331192], [-0.16552166640758514, 1.8883134126663208, -0.14542871713638306, -1.3940234184265137, -0.5174911618232727, 1.7682814598083496, 1.788816213607788, -0.3261301815509796, 0.5836125612258911, -0.7352079153060913, -0.00930654164403677, -0.41905784606933594, 0.06903044134378433, 1.064805030822754, 1.6597949266433716, -0.04608886316418648, -1.3601881265640259, -2.2034804821014404, -0.7868178486824036, -0.4647568464279175, 0.040885500609874725, -0.6486817598342896, 0.14286987483501434, 1.0719645023345947, 0.7136399745941162, -1.2570997476577759, -0.35450103878974915, -0.962513267993927, -0.3464086651802063, -0.4430285096168518, -0.5333846211433411, -0.47531819343566895, -0.2924887537956238, 0.43802428245544434, -0.8581013679504395, 0.5515471696853638, -1.1740458011627197, 0.05076818913221359, 0.7278255820274353, 1.405231237411499, 1.0631282329559326, -0.7846301198005676, 0.19055910408496857, 0.9100672602653503, 0.4497571587562561, -0.4254504144191742, 0.57977294921875, -0.22557130455970764, 0.5710861086845398, -0.45077452063560486, -0.5972647666931152, -0.10344748198986053, -0.38355183601379395, 0.2522924840450287, -0.7508253455162048, 0.5886275768280029, 1.0515239238739014, -1.216618299484253, 0.3096050024032593, 1.4926620721817017, 0.3294124901294708, -0.3145861029624939, -0.5014491677284241, -0.25242549180984497], [-0.18999503552913666, -0.7868058681488037, 0.19951924681663513, 1.1200242042541504, -0.8008759617805481, -1.2241538763046265, 1.2881523370742798, 1.0889723300933838, 0.4368562698364258, -0.4583110809326172, -0.12455446273088455, -0.47693055868148804, 0.2307758927345276, 0.9939225912094116, -0.009667097590863705, 0.12561458349227905, -2.6885199546813965, 0.8090470433235168, -0.8133827447891235, -0.3404405415058136, 0.5821719765663147, -2.0388970375061035, 0.44869908690452576, 0.21742969751358032, 0.9671738147735596, 0.4288114607334137, 1.2524032592773438, -0.6717556715011597, -0.7489376068115234, -0.6607192754745483, -1.5662981271743774, 0.05238451808691025, -0.8693913221359253, 0.5246997475624084, -1.2769184112548828, -0.8293067216873169, -1.3642042875289917, 0.4600350260734558, 3.358842372894287, -1.1913561820983887, -0.9467241764068604, -1.6051808595657349, -0.9237313866615295, -1.326963186264038, 0.9143140912055969, 1.1326977014541626, 0.6143469214439392, 0.9076799154281616, -0.8958559632301331, 0.26918286085128784, -0.9855098724365234, -0.7789320349693298, -0.4495920240879059, 0.4564259648323059, -1.1199992895126343, -0.08831775933504105, -1.0022279024124146, -1.4183884859085083, 1.2600061893463135, 1.5256421566009521, 1.0292459726333618, 0.5082200169563293, -1.6524349451065063, 0.4632307291030884], [-0.896845281124115, -1.3602228164672852, -1.277979850769043, -0.9690049290657043, 0.7665672302246094, 0.07573793828487396, 0.8837799429893494, -0.8664464950561523, -1.3423476219177246, 2.4007625579833984, 0.43650349974632263, 0.03889917954802513, 0.49032726883888245, -0.9346143007278442, 1.363138198852539, 0.041424233466386795, -1.1464754343032837, -0.9135313034057617, -0.7073329091072083, 1.013263463973999, -0.4608169496059418, 0.04248880222439766, 0.03820816054940224, -0.6671993732452393, -0.024666933342814445, 0.7739185094833374, -1.9281703233718872, -2.27127742767334, -2.206226348876953, 0.8207979798316956, -0.7892935872077942, -0.4407508671283722, -1.4552215337753296, -1.4636163711547852, 0.33776721358299255, -0.6307426691055298, -0.5281145572662354, -0.5129150748252869, 0.8437747359275818, -0.23060722649097443, 1.3092139959335327, 0.9010022878646851, -0.06538702547550201, -0.04660038650035858, 0.1104860007762909, 0.39559870958328247, -0.7060242295265198, 1.0540785789489746, 2.158690929412842, -0.34438779950141907, 1.8847341537475586, 0.44076985120773315, -1.0849390029907227, -0.018768899142742157, 0.33138102293014526, 0.529746949672699, -1.0405998229980469, 2.0869436264038086, -1.1468639373779297, 0.44461679458618164, 0.546558678150177, -1.2832506895065308, 0.3352588713169098, -1.6870442628860474], [1.3696272373199463, -0.836441695690155, 0.47399887442588806, -0.22463560104370117, -0.08704879879951477, 0.06935036927461624, 2.3233256340026855, -2.3142733573913574, 0.475943922996521, 1.5524406433105469, 2.755894422531128, -0.1813717633485794, -0.6928711533546448, -0.41305384039878845, 0.17445924878120422, -1.1606175899505615, -0.9168817400932312, 0.4039827585220337, -0.2128591239452362, -0.39425981044769287, -0.30954018235206604, 0.4240396320819855, 0.9207583665847778, 1.2804980278015137, 0.22886604070663452, 0.5954596400260925, -1.7078064680099487, 2.213507652282715, -1.1094815731048584, -1.4324920177459717, 2.3348000049591064, 1.4913326501846313, -0.3148830831050873, 0.7403126358985901, -1.2216730117797852, 1.230705976486206, 1.544559359550476, -1.208022952079773, -0.2728385031223297, 0.4037364721298218, 2.3185956478118896, 0.019194332882761955, 0.685927152633667, -1.4006017446517944, 1.0444430112838745, 1.7103495597839355, 0.004664508160203695, 0.1448279619216919, 0.9767526984214783, -0.0775231197476387, 1.2767306566238403, 0.1451246291399002, -1.1452049016952515, 0.1963154375553131, -2.5602006912231445, 2.30033016204834, 1.39732027053833, 1.4377084970474243, 0.7597129940986633, -0.11629659682512283, 1.8203562498092651, 0.5930150151252747, -0.7728840708732605, -1.4779635667800903], [-0.7298348546028137, -0.10368513315916061, -0.5324795246124268, 0.1604398488998413, -0.5416568517684937, 0.2710881531238556, -0.03224511817097664, -1.6004244089126587, 1.6379259824752808, 1.0664838552474976, -0.901902437210083, 0.04944575950503349, -1.0515868663787842, 0.9043657183647156, -0.6912917494773865, 0.21235373616218567, -0.38361266255378723, -0.9360920786857605, -0.4953007996082306, 1.106355905532837, -0.04182013124227524, 0.30844298005104065, -0.49128884077072144, 1.0540090799331665, -1.1725680828094482, -0.6948704123497009, 0.1704387366771698, 1.282898187637329, -0.5945910811424255, 1.474816083908081, 0.4846806228160858, -0.8698055744171143, -0.9351821541786194, -1.1863270998001099, 2.0246074199676514, 0.23658029735088348, 2.0101125240325928, 0.7591215372085571, 2.8784029483795166, 0.5175834894180298, -1.1156620979309082, 2.445855140686035, -1.4790483713150024, -1.1893914937973022, 0.05114975571632385, -0.11036485433578491, -0.9361022114753723, 0.42845579981803894, 0.5889337062835693, 1.2608364820480347, -1.3416991233825684, 0.2689765393733978, -0.11687805503606796, 0.4896232783794403, -0.5270270705223083, -0.38882797956466675, 0.16162389516830444, -0.13903558254241943, -1.9670777320861816, -1.0996553897857666, -0.7941816449165344, 0.7254392504692078, -0.3850572407245636, 2.136078119277954], [0.38004741072654724, 0.37468764185905457, 0.3484819531440735, 0.6262667775154114, -0.5971580147743225, 0.6117926239967346, -0.4247312545776367, 0.39664775133132935, 0.5401623249053955, -1.073562502861023, -0.30159831047058105, -1.1131258010864258, -0.5645734071731567, -0.3410487174987793, -1.3069453239440918, 0.3553659915924072, 1.2389775514602661, -0.2736729681491852, 0.3840497136116028, 0.8054289817810059, -0.8632632493972778, 0.22365552186965942, -0.5617551803588867, -0.21230444312095642, -1.1289516687393188, -0.42805618047714233, 0.5673393607139587, -0.8200587630271912, 0.433231920003891, 1.2258414030075073, 0.1984241008758545, -0.13007833063602448, -0.7796264290809631, 0.8902890086174011, -0.665989100933075, -1.3601462841033936, 0.02277953177690506, 1.6079707145690918, 1.426060438156128, -0.35448509454727173, 0.34788748621940613, -1.090465784072876, 1.1113680601119995, -1.9525970220565796, 0.4824731647968292, 0.26249954104423523, -0.7190461158752441, -2.743522882461548, 0.4042394161224365, -0.6802244782447815, 0.44010844826698303, 1.5608530044555664, 0.04783259332180023, -1.1017658710479736, -1.1181418895721436, -0.2577395737171173, -0.8272132277488708, 1.1624212265014648, -0.31809720396995544, -0.21731652319431305, -0.7826051115989685, 0.2907249927520752, 0.6812085509300232, -1.4068233966827393], [-0.6222691535949707, 0.6545444130897522, 0.1757168024778366, -1.0485150814056396, -1.0255721807479858, 0.6757948994636536, -0.14543017745018005, -1.2761926651000977, -0.7472324371337891, -1.3799152374267578, 0.010695849545300007, 1.7248272895812988, 1.9009591341018677, -0.2531373202800751, 0.7692251205444336, 0.8331355452537537, 0.8359174132347107, -0.41245025396347046, -1.8488922119140625, 0.743644654750824, -0.35478219389915466, -0.5704970955848694, -0.10433560609817505, -1.2205419540405273, 1.1974172592163086, 0.5686424374580383, 0.25638046860694885, 0.06294582039117813, -0.9351624846458435, -1.0961965322494507, 1.844029426574707, 0.9937928318977356, -0.5334099531173706, -1.707890510559082, -0.03560866042971611, 0.3331373333930969, -0.7881182432174683, -0.7464810013771057, 0.25440776348114014, -0.7798467874526978, -1.1666114330291748, 0.5929117798805237, 0.5258561372756958, 0.8114010691642761, 0.8290215134620667, 0.4519534111022949, 0.5320195555686951, -1.5775799751281738, 0.4142339527606964, -1.5257568359375, -2.201780080795288, -1.1094517707824707, -0.12126513570547104, 1.4128098487854004, -1.6465482711791992, -0.3223882019519806, -0.18771718442440033, -1.234387755393982, 0.19284671545028687, -0.519736647605896, 0.48648518323898315, 0.41276243329048157, 0.4287991225719452, 1.0043723583221436], [2.2141993045806885, -0.6212626099586487, -0.10942454636096954, 0.0572388619184494, -0.29825204610824585, 0.4498903155326843, -1.553044319152832, -0.4435783624649048, 0.049461159855127335, -0.33822619915008545, 0.09776610136032104, -0.5170382857322693, 1.2274326086044312, 0.09784504026174545, 0.5770567059516907, -2.039595603942871, 0.8001312613487244, 1.3363487720489502, 0.4419090747833252, 0.40329548716545105, -0.7673797011375427, -0.8786745667457581, -0.4704839885234833, 0.39983057975769043, 1.1121681928634644, -0.6381627321243286, -1.944071888923645, -0.19204367697238922, -1.3551422357559204, 0.42668601870536804, -1.8508260250091553, 0.5171979069709778, 1.213699460029602, 0.6401054859161377, -0.3767218589782715, 2.5845460891723633, -2.0834267139434814, -0.08816386014223099, -0.4296810030937195, -1.1065030097961426, -0.9996017813682556, -1.5122196674346924, 0.18346285820007324, 0.008088549599051476, 0.04185159504413605, -1.4515455961227417, 1.4994721412658691, -0.45722073316574097, -0.11535525321960449, -1.5616239309310913, -0.24046820402145386, -0.11970607936382294, -0.967174232006073, -0.19326038658618927, -0.18099601566791534, 0.7495685815811157, 0.48695746064186096, -0.34280163049697876, 1.1382695436477661, -0.6813114881515503, 0.43982794880867004, -0.645708441734314, 0.5040212273597717, -0.18014706671237946], [-0.5633772015571594, 0.2820728123188019, -0.18403133749961853, 1.417293906211853, -1.8338698148727417, -0.5539518594741821, -0.6238301992416382, -0.6823080778121948, 1.3342196941375732, 0.21616452932357788, -0.15479974448680878, -0.23550361394882202, -0.10565806180238724, -0.7076629996299744, -0.6224467754364014, 2.310431480407715, -0.6759684085845947, 0.5685298442840576, 0.005557267460972071, -0.9466666579246521, -1.4581809043884277, 0.9880761504173279, 0.5161811709403992, 1.7592171430587769, 0.23833513259887695, 2.016653299331665, -0.7584795951843262, 0.017400918528437614, 0.38690468668937683, -0.7248571515083313, 0.9788874983787537, 1.5868144035339355, 0.1911405622959137, 0.42731359601020813, 1.261520266532898, -1.2219089269638062, -0.17076608538627625, 0.2528461515903473, 0.3389885425567627, 2.917052984237671, -0.39333218336105347, -0.08233575522899628, 0.5670008063316345, 0.6575324535369873, 0.7183234691619873, -1.27891206741333, -0.3652976155281067, 1.9665558338165283, 0.5220673084259033, 0.7259317636489868, 0.266128271818161, -0.5372350215911865, 0.3932526409626007, -0.13940702378749847, 0.7652864456176758, -0.17973491549491882, -0.25086280703544617, -0.6449492573738098, -0.8313129544258118, -0.6846118569374084, -1.2656852006912231, -0.3766825497150421, 1.2354189157485962, 1.3086501359939575], [-1.3161604404449463, 0.6489256024360657, 0.12486287206411362, 2.2370190620422363, -1.1799918413162231, -0.28346773982048035, 1.3889771699905396, 0.6014769077301025, 0.8349671363830566, 0.3723716139793396, 0.0772080197930336, -0.2611091732978821, -0.12792415916919708, 1.7824461460113525, 0.45120954513549805, 0.2936866283416748, -1.7905436754226685, -1.0499922037124634, -0.10027389973402023, 1.5583008527755737, -0.11823826283216476, -1.0331158638000488, 0.9963459372520447, 0.0022822695318609476, 0.27873045206069946, 2.547309637069702, 0.5691553950309753, 0.5957416296005249, 1.002333164215088, 0.5610800981521606, -0.17486174404621124, -0.43112820386886597, -0.18198281526565552, 0.10632966458797455, -1.1361747980117798, -1.4397112131118774, 0.8979447484016418, -0.30140215158462524, 0.9232169985771179, -1.0417077541351318, 1.63160240650177, -0.8958191275596619, -0.2590401768684387, 0.5349862575531006, 0.17834295332431793, -1.592271089553833, -1.393298864364624, 0.6993873715400696, -0.10005918890237808, 0.8796459436416626, 2.2525105476379395, 0.5026148557662964, -0.14868229627609253, 0.17734532058238983, 0.25104591250419617, 0.1098090261220932, 0.09480056911706924, 0.4624478220939636, -0.14267118275165558, 0.13264277577400208, 0.2841791808605194, -0.3094506561756134, -1.3556095361709595, 1.1851001977920532], [-0.3567470908164978, 0.7480504512786865, -0.8263989090919495, -0.012234401889145374, -0.4926301836967468, 1.008849859237671, -1.51048743724823, 0.41310781240463257, 1.5607097148895264, -0.5184895992279053, 1.1069117784500122, 1.6454194784164429, 0.9075493216514587, -0.2981136441230774, 1.194261074066162, 0.6813585162162781, -1.487047791481018, -1.0980160236358643, 1.4727007150650024, 0.605339765548706, 0.7194694876670837, 0.2069968432188034, -0.4754275381565094, 0.437091588973999, -1.3746534585952759, 2.087714672088623, 2.549997568130493, -2.798429489135742, 1.524657130241394, 0.17463412880897522, 0.09990378469228745, 1.1360920667648315, -1.2169135808944702, 0.42681124806404114, 1.8193070888519287, 1.0792604684829712, 2.9537618160247803, 0.425586074590683, 0.24553139507770538, 0.9830585718154907, -1.4368069171905518, 0.171066015958786, 0.4878132939338684, 1.532481074333191, 0.5820502638816833, 1.1151847839355469, 0.8782334327697754, 0.12725117802619934, 0.25731155276298523, 2.421349287033081, 1.2451239824295044, 0.7894918918609619, 1.429760456085205, -0.9824773669242859, 0.4063621163368225, -1.0176712274551392, 0.9720278978347778, 0.44003239274024963, -1.254525899887085, 0.21522046625614166, 1.3371083736419678, 1.3689348697662354, 1.071635127067566, -1.2052185535430908], [2.2334930896759033, 1.3146395683288574, 0.9768478870391846, -0.9665055871009827, -0.012380659580230713, -0.4361603260040283, 1.0219603776931763, 1.128535509109497, -0.1381293684244156, 0.5356746912002563, 2.410050392150879, 0.8340997099876404, -0.06527993083000183, 0.08839159458875656, 0.22519367933273315, -0.43560469150543213, 2.1435298919677734, 0.21102195978164673, -0.14978483319282532, 1.3582041263580322, -0.6189965009689331, -0.6718677878379822, 0.5677880048751831, 0.07500437647104263, 0.7197574377059937, 2.2290241718292236, -0.539819598197937, 0.3071128726005554, -1.2673516273498535, -1.328874111175537, 0.14254096150398254, -0.6864930987358093, -0.4959244430065155, 1.3728963136672974, 2.3793296813964844, 0.2793242037296295, -0.21693703532218933, -1.4169615507125854, -0.7058699727058411, 0.595020055770874, -1.741241216659546, 0.14154905080795288, 0.7757296562194824, 1.6415798664093018, -0.29488295316696167, 0.7257992029190063, -1.368111252784729, 0.3929157257080078, -0.024738715961575508, 0.23361670970916748, -0.7389054298400879, -1.335474967956543, -1.530508041381836, -0.41775110363960266, 0.9037945866584778, -0.3314095139503479, -0.5367668867111206, -2.1548733711242676, -0.5850980877876282, -0.5785418748855591, 0.7266827821731567, 2.3849856853485107, -2.381382465362549, 1.221494197845459], [0.7014304399490356, 1.2171813249588013, -0.17495097219944, -0.26558321714401245, -0.9073450565338135, 1.4720470905303955, -0.9468643665313721, -0.1703081876039505, 0.33979740738868713, -1.4806874990463257, -1.317704439163208, -0.9753329157829285, 1.6093685626983643, -1.397599220275879, -0.4100603461265564, -0.2757972478866577, 0.8182750940322876, -1.05897855758667, 0.5492727756500244, -2.15244722366333, 1.0267298221588135, -0.6797064542770386, 0.1063450276851654, -0.5123116970062256, -1.3239576816558838, 1.1341451406478882, -0.3333226144313812, 1.0785207748413086, 0.737614631652832, -1.2366713285446167, 0.23184150457382202, -0.7463943958282471, -0.1776702105998993, 1.106562852859497, 0.7038820385932922, -0.8611679077148438, 0.751085638999939, -0.37787336111068726, 2.5578818321228027, -0.29648566246032715, 1.2395122051239014, -0.2961898148059845, -0.8577888011932373, 0.6932159066200256, 0.14493995904922485, -2.188872814178467, -0.8709763288497925, -1.0063732862472534, 0.21038906276226044, 0.0341557040810585, -0.022933222353458405, -0.09726199507713318, -0.2509588599205017, -0.22350814938545227, -0.13619059324264526, -0.15488067269325256, -0.08755438029766083, -1.3204402923583984, -0.18106862902641296, 0.49236488342285156, 0.7361283898353577, -0.11337278038263321, 0.9645583033561707, -0.7719467282295227], [-0.40912100672721863, 0.06265882402658463, -2.615607976913452, -0.3524360656738281, -0.9775373935699463, -0.8395878672599792, 0.8358615636825562, 1.0766853094100952, 0.10772831737995148, -0.611370861530304, -0.3146880865097046, 0.061356354504823685, -1.091704249382019, 1.294721245765686, -0.11566408723592758, -0.943132221698761, 0.7609307765960693, -1.373272180557251, -0.412056028842926, -0.8287702798843384, -0.4303453862667084, 1.8783559799194336, 0.9784500598907471, -0.2454010397195816, 1.0320394039154053, -0.2904261350631714, -0.24306367337703705, -0.6515914797782898, 0.9555543661117554, -1.131838321685791, -0.7294321060180664, 0.5510597825050354, 1.1211698055267334, -2.0398566722869873, 0.7123904824256897, -1.2282190322875977, -1.1132813692092896, 0.8560581803321838, 1.7118910551071167, 1.3623790740966797, -1.9871119260787964, 0.050247009843587875, 0.23540323972702026, 1.2102919816970825, 0.7960973381996155, 2.6467299461364746, -1.1127469539642334, 0.014585469849407673, 0.05168382078409195, 0.25088584423065186, 0.7095807790756226, 0.5324967503547668, -1.0993667840957642, 0.2375684231519699, -0.42999544739723206, -1.060302495956421, -1.2305420637130737, 0.16385087370872498, -0.5319169163703918, 1.2767318487167358, 0.13183143734931946, 0.2348114550113678, 0.49316489696502686, 1.2926090955734253], [-0.8450076580047607, 1.6232872009277344, -0.45322585105895996, 0.12303545325994492, 0.2539781630039215, -1.031914234161377, -0.6656161546707153, -0.30390581488609314, -0.7604756951332092, -1.1366840600967407, 0.27652600407600403, 2.7168872356414795, 1.335928201675415, 1.282964825630188, 1.6799565553665161, 0.2604048252105713, -1.1046619415283203, 2.7445483207702637, 0.2453901469707489, -1.6824743747711182, 2.5008156299591064, -0.2107148915529251, 0.8599920272827148, 0.9669017791748047, 0.5030171275138855, -0.7863643765449524, 0.40895017981529236, 0.9733723998069763, 0.8495391011238098, 0.7652848958969116, 0.09413354843854904, 0.5901919007301331, 0.7096648812294006, -0.7616689205169678, 0.5319104194641113, 0.679168164730072, -1.501416802406311, 0.3253583312034607, 0.06029195338487625, -0.3018626272678375, -0.8833286166191101, -0.4723953902721405, -0.1725432127714157, 1.549767017364502, 1.7232898473739624, 1.2598086595535278, 0.8445464372634888, -1.9044945240020752, 0.7508403062820435, 0.04101188853383064, 0.00026836286997422576, -0.09221845120191574, -0.652296245098114, -0.250498503446579, -1.6001180410385132, 1.0937221050262451, -0.9029802083969116, -0.9952994585037231, 0.9617586731910706, -1.9324212074279785, 0.14164869487285614, -0.4514486789703369, 1.1687986850738525, -1.749192237854004], [0.6761971712112427, -0.7957623600959778, 0.5984198451042175, -0.4021865129470825, 1.0837182998657227, -0.7683802247047424, 1.3127219676971436, -0.46786537766456604, -0.13795900344848633, -1.1718956232070923, -0.7772125601768494, -0.02572743594646454, -0.743961751461029, 0.29124757647514343, 1.0420382022857666, 0.2108469307422638, 0.9469642639160156, 0.13059139251708984, 0.10758968442678452, -1.5256887674331665, -0.366840124130249, -0.3189135491847992, -0.14801478385925293, -1.103903889656067, -0.7297456860542297, -1.395210862159729, -2.2112646102905273, -2.0942561626434326, -0.20156675577163696, 0.9051222801208496, -2.400524854660034, 0.14102861285209656, 1.3467785120010376, -1.6292701959609985, 1.3119752407073975, 0.7415692806243896, -0.44806545972824097, -0.551078200340271, 0.4976767897605896, 0.1303527057170868, 0.34567922353744507, -0.5674781799316406, -0.32243427634239197, 0.6189442873001099, 0.22409503161907196, 0.11855897307395935, -0.4965224862098694, 0.892939567565918, 0.19651229679584503, -1.652808427810669, 0.6300212144851685, 2.3535103797912598, 0.768427848815918, -0.6222286224365234, -1.1011689901351929, -0.7596635222434998, 0.8476241827011108, 0.8655759692192078, 0.36559683084487915, -1.1057559251785278, -2.919574737548828, 0.8003477454185486, 1.8209375143051147, 1.3703182935714722], [0.5275551080703735, -1.6882554292678833, 1.0530325174331665, 0.8941375613212585, -0.3158690333366394, -0.08372107148170471, 0.1651030331850052, 0.18811820447444916, -0.5728272795677185, -3.7654330730438232, 0.4954942762851715, -0.6167969703674316, -0.5037741661071777, -0.04974598065018654, 0.1957819163799286, -0.7085688710212708, 2.554037570953369, -1.1167480945587158, 0.20015913248062134, -0.8493167757987976, -1.1279677152633667, -0.7673422694206238, 1.3391146659851074, 0.024455919861793518, -1.7363641262054443, -0.4925794005393982, -1.2813814878463745, -0.3150481581687927, 0.7269381880760193, 0.5167455673217773, 0.6982098817825317, -1.4074499607086182, -0.4131699204444885, 0.46049633622169495, 0.6478679180145264, 0.12034347653388977, -0.27192771434783936, 0.9954133629798889, 2.5314714908599854, 0.9593555331230164, 0.022905554622411728, -0.001826582127250731, 0.7111272215843201, 1.8270795345306396, 0.23935601115226746, 0.20700578391551971, 0.7099606990814209, -0.9156070351600647, -1.1443349123001099, 1.6103006601333618, -0.03383781388401985, 0.986778974533081, -2.695021390914917, -1.251106858253479, 0.15781085193157196, 0.242247074842453, -0.10281680524349213, 0.5229335427284241, 0.407732218503952, 0.9574706554412842, -0.240413635969162, -1.0345648527145386, 1.237280249595642, -1.153922200202942], [-1.0544769763946533, -0.3597027063369751, 0.48597365617752075, -0.9431750774383545, 0.6782896518707275, 0.9840412139892578, 1.2370041608810425, -0.018580079078674316, -1.94333815574646, 1.4411476850509644, -0.11059918999671936, 2.0233521461486816, -0.4153240919113159, -0.1199834942817688, -1.1390328407287598, -0.9596803188323975, -0.3846515715122223, -0.39914670586586, 0.2255133092403412, -0.6650274991989136, 0.7920702695846558, 0.8063021898269653, -0.9952926635742188, -0.11329822242259979, 0.36224982142448425, 0.2666873037815094, 0.02770061232149601, 0.08772100508213043, 0.10912416875362396, -0.2925882339477539, -0.06210707128047943, 2.4853148460388184, 0.3395257294178009, -1.311426043510437, 1.732336401939392, -0.9454737901687622, -0.7428348064422607, -0.27130386233329773, 0.062340229749679565, 0.9995840787887573, -0.044814642518758774, -1.8349062204360962, -0.03766308352351189, 0.31031155586242676, -1.0682075023651123, 1.0177536010742188, -0.9650226831436157, -1.029486060142517, 0.04230460152029991, 2.243703842163086, 1.666945219039917, 0.05474957078695297, -0.15969127416610718, -0.4598071873188019, -0.5824594497680664, 1.306652307510376, -0.058107271790504456, 1.4446310997009277, -0.10482832044363022, 0.2239934206008911, 0.18063245713710785, 0.7166273593902588, -0.9734151363372803, 1.044662594795227], [-0.09283128380775452, -0.35539084672927856, -1.1868141889572144, 0.5744296312332153, -1.2882872819900513, -2.0680267810821533, -0.16406391561031342, 1.1817395687103271, -0.056200094521045685, -0.22330167889595032, -0.07907599210739136, 1.0303736925125122, -0.8182457089424133, 0.1228523850440979, -1.1398611068725586, 1.6862396001815796, -0.6421111822128296, 1.5892547369003296, 0.009663661941885948, -0.18100328743457794, 0.758270263671875, 0.6630631685256958, -1.109702229499817, 1.3151975870132446, 1.3637886047363281, 2.455109119415283, -1.2868337631225586, 0.2731269598007202, 2.023759365081787, 0.9090170860290527, 1.2882459163665771, -0.4892174005508423, 1.5541186332702637, 0.6023495197296143, 0.4920780062675476, 0.42422446608543396, -1.0428762435913086, 0.7522485256195068, -0.3971545696258545, -0.30254414677619934, 0.7377200722694397, 2.907304048538208, 0.4761047065258026, 1.3147907257080078, 0.008320887573063374, 1.2822662591934204, 0.7389675378799438, -0.839815616607666, -1.2556161880493164, 0.32305407524108887, 1.1296958923339844, -0.4135538935661316, -1.5702167749404907, 0.268937349319458, -2.487307548522949, -1.3504552841186523, -0.21359997987747192, -0.6663380265235901, 0.00947294570505619, -0.4563647508621216, -0.1974955052137375, 1.2317543029785156, -0.06457500159740448, -0.2176395058631897], [-1.3251622915267944, 2.163259744644165, 0.41809406876564026, -0.7383710741996765, 0.23786285519599915, -1.246717095375061, 1.0907800197601318, 1.3516464233398438, -0.5302179455757141, 0.31080856919288635, -0.3229513466358185, 0.4471486508846283, -0.967694103717804, 0.9961516261100769, -0.20515123009681702, -1.5770090818405151, 1.012406349182129, -1.0548591613769531, 0.21076041460037231, 2.025834321975708, -2.884143352508545, -0.5348633527755737, 0.18467994034290314, 0.05022398754954338, -1.68888258934021, -0.5299888849258423, 1.0121537446975708, 0.38232535123825073, 2.642219066619873, 0.6179251670837402, 1.4956464767456055, -0.7074670195579529, 0.3311871886253357, 1.0094472169876099, -0.723732054233551, 0.33784547448158264, -0.40411412715911865, 0.47566625475883484, -0.3468921184539795, -0.20341819524765015, -0.1086440235376358, -0.36961352825164795, -0.2722916603088379, -0.46493518352508545, 0.5233938694000244, -0.04066791012883186, -1.7434284687042236, 0.06266258656978607, 0.38289862871170044, -0.4766990840435028, -0.3457416296005249, 1.6710505485534668, -0.6884268522262573, 0.9836322069168091, -0.45630699396133423, 0.05394332855939865, -1.4230717420578003, -1.9632364511489868, 0.5809728503227234, 2.1626877784729004, -1.1247535943984985, 1.85772705078125, -0.026671260595321655, -0.5639535188674927], [1.0705024003982544, -0.8164457082748413, 1.2205826044082642, 1.2612028121948242, -1.4245038032531738, 0.23277394473552704, 0.8639212846755981, 0.5878519415855408, 1.67302668094635, 0.6956840753555298, 0.060355041176080704, -0.010133457370102406, 0.08824009448289871, 0.19948594272136688, 0.7669218182563782, 0.8982418775558472, -0.8196988701820374, 1.0738534927368164, -0.2762623429298401, -0.2710287868976593, 0.2961564362049103, -1.3273091316223145, 0.8508515954017639, 0.45610302686691284, 1.1152654886245728, 1.5772453546524048, -3.3213725090026855, -1.452730417251587, -0.44972071051597595, 0.6961468458175659, 0.6240566968917847, 0.6363651752471924, -0.9024007320404053, 0.7880438566207886, 0.49088239669799805, -0.2624705731868744, 0.8177147507667542, -1.1753180027008057, -0.5823164582252502, 1.7789506912231445, -0.2840668261051178, -0.8117102384567261, -0.8207542300224304, -1.8249200582504272, -1.5141880512237549, -0.3188536763191223, 1.1800224781036377, 0.04614303633570671, 1.049905776977539, -0.8841358423233032, 0.05463043600320816, -1.1767818927764893, -0.07352253049612045, 0.611443281173706, -0.2819572389125824, 0.24560381472110748, 0.762182891368866, -1.004196286201477, 1.6196093559265137, 0.6234614253044128, -0.7594596743583679, -0.19207295775413513, -1.596605658531189, -0.19825662672519684], [-0.15377628803253174, 0.5807462930679321, 0.0061726863496005535, -1.2729436159133911, -0.4183961749076843, 1.5216861963272095, 0.7950268387794495, 0.651371955871582, -0.4296797513961792, -1.5831400156021118, 0.1350499987602234, -0.7876360416412354, 0.43099790811538696, 0.09653910994529724, 0.4007362425327301, 0.5718020796775818, -0.26245856285095215, 0.9829354882240295, -0.3605769872665405, 2.138662576675415, 0.8610652685165405, 0.5653780102729797, -0.369153767824173, 0.6865717172622681, -0.5538707375526428, 0.5847684144973755, -1.5357861518859863, 0.6170251369476318, -0.31608814001083374, 0.283143013715744, -0.3072153627872467, 1.2358444929122925, 2.9489777088165283, 1.4051666259765625, -0.11279000341892242, -1.1615228652954102, -1.6211892366409302, 2.1342668533325195, -0.6598566174507141, -0.6033117175102234, -0.9089422225952148, -0.3229161500930786, 0.04027353599667549, -0.1529063731431961, -0.08825360983610153, 0.27408677339553833, 0.6281046271324158, -1.3930561542510986, -0.1767619401216507, 0.05485357716679573, 0.48319411277770996, 0.10118713229894638, 0.9983892440795898, -0.6171693205833435, -0.6636871099472046, 0.39524996280670166, 0.6095097661018372, 0.4301307797431946, 0.17745517194271088, 0.11722132563591003, -0.4366765022277832, -1.3100720643997192, -0.7764647006988525, -0.5030261278152466], [-1.7059041261672974, -0.6167227625846863, 0.19375406205654144, -0.6172563433647156, -0.1399756222963333, 0.31388092041015625, -0.20243991911411285, 1.9588640928268433, -0.033524829894304276, -0.1085902750492096, 0.3542163372039795, -0.20970401167869568, 0.6865691542625427, 1.2349933385849, -0.015413929708302021, 0.9205535054206848, -2.182251453399658, -0.6997338533401489, 0.20578710734844208, -0.22452104091644287, -1.9130395650863647, -0.19611810147762299, -1.3815442323684692, -0.15653999149799347, 0.5391306281089783, 1.1067924499511719, -1.5314890146255493, 0.5355282425880432, -0.7001932263374329, -1.4026943445205688, 0.24830704927444458, -0.018985148519277573, -1.657407522201538, -0.05146893486380577, 0.20666182041168213, -2.081547975540161, -1.3731600046157837, -0.7436495423316956, 0.9890403151512146, -0.5851812362670898, 0.6839040517807007, 0.3943406641483307, -1.2868597507476807, 2.1743698120117188, 0.1880020946264267, 0.00017335107258986682, 0.18395216763019562, -1.652365803718567, -0.14561231434345245, 0.919955849647522, -0.1324392557144165, -0.7386439442634583, -0.6738502383232117, -1.461809515953064, -1.193057894706726, 1.3081600666046143, 0.690074622631073, 1.1637983322143555, -0.7251927852630615, 0.8608933687210083, 0.14422746002674103, -0.8417436480522156, -0.8159010410308838, -1.7710765600204468], [1.349957823753357, 1.019906759262085, -2.580876588821411, 1.8397326469421387, 0.33553630113601685, 0.022447697818279266, -0.550051748752594, -1.7591367959976196, 0.8138924837112427, -0.053354308009147644, 0.10321284830570221, 0.16641002893447876, -0.6026305556297302, 0.5160616636276245, 0.23114873468875885, 1.7907545566558838, -2.025101661682129, -0.03512866422533989, 1.0622401237487793, 0.5679787993431091, 2.909331798553467, -0.2604033946990967, 0.41884249448776245, 0.07668137550354004, -1.3810696601867676, -0.44514745473861694, 1.8305840492248535, 0.41989049315452576, -0.035847850143909454, -0.22032231092453003, -0.18976782262325287, 1.4508299827575684, -0.34462326765060425, 0.9997850060462952, 1.2386237382888794, -0.2875363826751709, 1.3729910850524902, -1.043657898902893, -0.715559184551239, 1.0132472515106201, 0.7950252294540405, 0.9669795632362366, -0.9378131031990051, 0.22706972062587738, -1.2520062923431396, 0.5782604813575745, -0.7123304605484009, -1.1968612670898438, -1.351418375968933, -1.0706406831741333, -1.0024346113204956, 0.48856106400489807, -1.3204193115234375, -1.7684084177017212, -0.5421980023384094, 1.1758294105529785, 0.931242823600769, 0.7257710099220276, -2.348536491394043, 0.35141927003860474, -0.7909619212150574, 0.4923189878463745, 0.16634991765022278, -0.29915565252304077], [-0.30564117431640625, -1.1141880750656128, 0.5007301568984985, -2.1430306434631348, 0.44733119010925293, 0.20769110321998596, 0.22172822058200836, -0.24278831481933594, 0.3789777159690857, 0.9253275990486145, -0.08598145842552185, -0.49476897716522217, -0.7583884596824646, -1.4397904872894287, -2.144804000854492, 0.030352899804711342, 0.3229091465473175, -0.2592197060585022, 0.4017772078514099, -0.9255077242851257, 0.852911114692688, -2.394185781478882, 0.4650557041168213, -1.596060037612915, -1.4666650295257568, 3.126908302307129, 0.26343855261802673, -0.7436017394065857, -0.9044255614280701, 0.1820298433303833, 0.8883703947067261, 0.32376646995544434, 0.9523910880088806, -1.2401256561279297, -2.4474685192108154, 0.3950033187866211, 0.917890727519989, -0.8397253751754761, 1.6728426218032837, 0.2376122921705246, 0.7056340575218201, -0.5734902620315552, -2.189047336578369, -0.6473541855812073, 1.2063523530960083, 0.6271705031394958, 0.7899979948997498, -0.9070459008216858, 0.3462517559528351, 1.7301814556121826, 0.41863781213760376, 0.42050105333328247, -0.018502887338399887, -0.9944988489151001, -0.2357206493616104, 0.7706934213638306, 0.9289558529853821, 0.21597284078598022, -1.0781804323196411, 1.6404896974563599, 1.359236717224121, -1.247421145439148, 0.21699708700180054, 0.4476030468940735], [1.1230679750442505, 0.24469265341758728, -0.8648287057876587, 0.13269275426864624, -0.22114135324954987, -1.4610809087753296, 1.1425609588623047, 0.179734006524086, -3.0420312881469727, 1.570608139038086, -0.5612019300460815, 0.10557752102613449, -1.5954029560089111, 0.474031537771225, -0.29723450541496277, 0.02633132040500641, 0.884102463722229, -2.3441193103790283, -0.35921117663383484, 1.640650987625122, -1.0778290033340454, 1.067192792892456, -0.5447671413421631, -1.029033899307251, 0.32166749238967896, 0.10209069401025772, 1.6277151107788086, -0.15795281529426575, -0.4131443500518799, 0.6528711915016174, -1.1122239828109741, -1.054579496383667, 0.22681570053100586, -0.9639395475387573, -1.6416019201278687, -1.0871678590774536, -0.3771388530731201, -0.6921061873435974, -1.5080981254577637, 0.6722765564918518, -0.04530435428023338, -0.10614005476236343, -0.2859686017036438, 1.412805438041687, -0.2626839280128479, 0.49968260526657104, -1.1816152334213257, 0.3036247491836548, -0.6754024624824524, 1.295501470565796, -0.8520800471305847, 1.4498114585876465, 0.18532569706439972, 0.5054683685302734, 1.8525722026824951, 0.007271865848451853, -0.07506854087114334, 1.165942668914795, -0.8149996399879456, -0.23233428597450256, -0.43195581436157227, 0.32212114334106445, 0.4103490710258484, -0.13015209138393402], [-0.12088315188884735, 1.6773502826690674, -0.2491065412759781, 0.7046014070510864, 0.9596978425979614, -0.004211636260151863, 3.3116869926452637, -0.7554241418838501, -1.573636770248413, 1.5504103899002075, 1.495686650276184, 1.1293917894363403, -0.34351417422294617, 0.05179969593882561, -0.07087340950965881, -1.669791579246521, -0.38819780945777893, 0.7848276495933533, 0.682485818862915, -0.11697361618280411, -0.15285028517246246, 0.10126838088035583, -0.9872223734855652, 0.43161970376968384, -0.714363694190979, -0.7273328900337219, -1.1218042373657227, -1.2045334577560425, 0.9060928225517273, 0.14873439073562622, -0.5431457757949829, -1.57033109664917, -1.3456802368164062, -0.6283173561096191, 1.3261293172836304, -0.4556679129600525, -1.040244698524475, -0.5577453970909119, -0.3808169364929199, 1.2358566522598267, -1.8770734071731567, 0.00808838102966547, -0.24684812128543854, 0.6954964399337769, 1.742255687713623, 1.1837542057037354, 2.6014161109924316, -1.436974287033081, 0.6369725465774536, -0.24605029821395874, -1.0576666593551636, 0.3672105371952057, 0.7030320763587952, 1.2050518989562988, -3.6743268966674805, -1.6095272302627563, 1.0884087085723877, 1.2285594940185547, -1.6433483362197876, 0.5551807880401611, -0.6570979952812195, 1.1466487646102905, -0.5594347715377808, -0.3356969356536865], [0.18706072866916656, -0.15501534938812256, -1.2971018552780151, -0.8147076964378357, -0.39501848816871643, -0.2258237600326538, 1.1100611686706543, -2.3927528858184814, 0.21465064585208893, -1.687857747077942, -0.964417040348053, -0.6815560460090637, 0.7644349336624146, 0.049785129725933075, -0.8376865983009338, -0.2612764537334442, -0.05247613415122032, 0.15609639883041382, -1.7312747240066528, -0.025035643950104713, -0.2013026624917984, -1.4117522239685059, -1.3572274446487427, -0.10760930180549622, -0.45201149582862854, -0.50691819190979, 0.29843345284461975, -0.11419063061475754, -0.6052956581115723, 0.3444000482559204, -0.42609402537345886, 1.5984818935394287, 1.7911686897277832, -1.0367053747177124, -0.9032283425331116, 0.6792353391647339, 1.6952180862426758, 2.7697486877441406, 1.9300031661987305, -0.17792895436286926, -1.2324692010879517, 0.7917731404304504, 0.5016322731971741, 0.4392290711402893, 1.2567780017852783, -0.22013497352600098, 1.5372391939163208, -0.6938967704772949, 0.6557767391204834, 0.5427541732788086, 1.534271001815796, 0.0787372887134552, 1.04096519947052, 0.768053412437439, -0.01784176006913185, -0.3905792534351349, 0.7725549340248108, -1.7505167722702026, -0.5712526440620422, -1.6683796644210815, 0.8576545715332031, -0.20773419737815857, -2.388918161392212, -0.03066619671881199], [-0.8955013155937195, -1.3914847373962402, -1.3635427951812744, 0.7602645754814148, -0.38399332761764526, 0.648879885673523, -0.1654762625694275, -0.8838410973548889, -0.22621384263038635, -0.712291955947876, -0.8828945755958557, -0.37095507979393005, -0.4278891682624817, -0.10217280685901642, 1.0613951683044434, 0.24314424395561218, -0.5600446462631226, 0.015825293958187103, -1.163778305053711, -0.036866821348667145, -1.9586752653121948, 0.8149274587631226, 1.092448353767395, -1.9085872173309326, 0.6057283878326416, -0.07803089171648026, -0.12623748183250427, 0.3238644301891327, -1.2022641897201538, 0.43876245617866516, 0.12136341631412506, 0.05307023227214813, 0.11230236291885376, 0.26801058650016785, -1.7781492471694946, -2.1176199913024902, 0.7887601256370544, 1.4240496158599854, 0.3058270812034607, -0.8147991299629211, -0.9381239414215088, 0.052805088460445404, -0.4229866862297058, -0.3147814869880676, 0.2869817018508911, 1.280334711074829, 0.9491477012634277, -0.49892479181289673, -1.478047251701355, 0.03337995707988739, 0.8556475043296814, -0.026789039373397827, 0.43457499146461487, -1.7831287384033203, 0.8660477995872498, 0.18066319823265076, -0.31056898832321167, 0.7498654127120972, 0.479937881231308, 0.4265006184577942, 1.0727416276931763, -0.0077820029109716415, -0.028867008164525032, 0.13828928768634796], [-0.6970154643058777, -1.1398611068725586, 1.535586953163147, 0.7150053977966309, 0.15468786656856537, -0.17902463674545288, 0.79514080286026, -0.8516332507133484, -1.2757054567337036, -0.46066662669181824, 0.657755970954895, 0.06398563832044601, 2.226175308227539, -0.2991976737976074, -0.5259028673171997, 1.6895643472671509, -0.7393686771392822, 1.070168375968933, -1.2657270431518555, 0.27161043882369995, 1.1468147039413452, -1.1768792867660522, -0.8848448991775513, 0.3698861598968506, -0.24121272563934326, -0.08718083798885345, -0.4685511291027069, 0.41867101192474365, -2.7967679500579834, 1.0946080684661865, 0.10717363655567169, 1.322827935218811, -1.0936492681503296, 1.9173579216003418, 0.23717515170574188, 1.266921043395996, -0.6766720414161682, 0.05060186982154846, 0.8977776765823364, 0.5618146657943726, -0.6137688755989075, -1.6908682584762573, -1.8530113697052002, -0.504564106464386, -0.34546083211898804, -0.3469776511192322, 1.3960260152816772, 0.14560729265213013, -0.08337100595235825, 0.3588505983352661, 0.6521424055099487, 0.8200078010559082, 1.2672510147094727, 0.8376340866088867, -0.6180509924888611, 1.4608871936798096, -0.5844129323959351, 0.7284131646156311, -0.9846190810203552, 0.5156156420707703, 0.4952622056007385, 0.22257235646247864, 2.104414939880371, -1.3961721658706665], [-0.039561714977025986, -1.163399577140808, 0.43749839067459106, 0.2877067029476166, 0.0695728063583374, 0.37348347902297974, 1.4283151626586914, -1.0502043962478638, 0.42004451155662537, 1.1349132061004639, 0.2563709616661072, -0.020331665873527527, 1.5526611804962158, 0.305962473154068, 1.276139259338379, -2.5819575786590576, -1.9060667753219604, -1.0320078134536743, -0.9969145059585571, 1.2478787899017334, 0.7765533328056335, -0.4538255035877228, 0.6145133972167969, 0.8576601147651672, 1.6009349822998047, 0.9150134325027466, 0.23420554399490356, 0.5488932132720947, 1.2496320009231567, -0.8558814525604248, 0.05121413618326187, -0.7961440682411194, -0.26036936044692993, -0.6713098287582397, 0.23416323959827423, -1.5976200103759766, 1.3159947395324707, 1.0961421728134155, -0.5634576678276062, 1.1979466676712036, -0.9324145317077637, 1.907414197921753, 1.3182333707809448, -0.6468089818954468, -0.5678825378417969, 1.0090224742889404, -1.2949467897415161, -1.7462819814682007, -0.8121973276138306, 0.2267885059118271, 0.7939101457595825, 0.10967744141817093, -0.5574195981025696, 2.221705913543701, 0.35262274742126465, 0.8673593401908875, 0.679987907409668, -2.192457437515259, 1.3942292928695679, -1.8073474168777466, -0.24161511659622192, -1.4855374097824097, 0.36302587389945984, 0.8890871405601501], [-0.47822484374046326, -0.5355815887451172, -0.6954165697097778, 1.1049809455871582, -1.2501254081726074, 1.2791708707809448, 0.6884972453117371, 0.44286102056503296, -0.6506499648094177, -0.4343624413013458, 0.7847518920898438, -0.6403905749320984, 0.7818995118141174, 0.12178648263216019, 0.7443525791168213, 0.5506888628005981, -1.0878790616989136, -0.23928366601467133, -1.4777783155441284, -0.09193943440914154, -0.6026198863983154, -0.16979196667671204, -0.8036059737205505, -1.6077215671539307, -0.7520323395729065, 0.7710970044136047, -0.3383640646934509, -0.853711724281311, 0.09078819304704666, 0.47391530871391296, 0.9047778844833374, -2.772778034210205, 1.2096291780471802, 0.4363804757595062, -0.3111365735530853, 1.9708282947540283, 1.9206178188323975, -0.883542537689209, 0.5070092082023621, -2.0938940048217773, -1.3149118423461914, -0.9342392683029175, 0.397749662399292, 0.21011459827423096, 0.41032591462135315, 0.4091801345348358, -0.11042623221874237, 1.0493851900100708, 0.543597936630249, 0.23321165144443512, -1.8473371267318726, 1.1709426641464233, -1.5125000476837158, 0.5702629685401917, -1.0875757932662964, -0.4966275990009308, 0.897981584072113, 0.4170977771282196, -1.196171522140503, 0.0890372171998024, -1.9860618114471436, -0.21408246457576752, 0.16886556148529053, 0.7713248133659363], [1.1058052778244019, 0.6611353158950806, -0.9058699011802673, -0.04037243500351906, -1.0581738948822021, 1.805922508239746, -0.7596864700317383, 0.34731027483940125, 0.37660446763038635, 2.151217222213745, -0.44879403710365295, -0.022898273542523384, 1.777870774269104, 0.028352363035082817, -0.02674272656440735, -0.0644649863243103, 3.1158597469329834, 0.5533053278923035, 0.6184849143028259, 0.8510400056838989, -0.8267194628715515, 0.489775687456131, -1.35361647605896, 0.3743862211704254, -1.1587358713150024, 0.367218017578125, -0.2919010818004608, 1.1403998136520386, 0.41560909152030945, 0.9241821765899658, -0.7919843792915344, 1.0123835802078247, 0.4242074191570282, 0.14928196370601654, -0.24592891335487366, -0.7642231583595276, 0.32922521233558655, -0.33912500739097595, 0.04518691822886467, 0.5965545177459717, 0.7298374772071838, 0.8065161108970642, -0.5509238243103027, 0.3174475431442261, 2.1159629821777344, 1.2057102918624878, -0.012849588878452778, 1.0801129341125488, 1.0013306140899658, -0.36611053347587585, -0.630516767501831, 1.2403323650360107, -1.1849265098571777, 0.6685911417007446, 0.41578125953674316, 0.15471118688583374, 1.8087509870529175, -0.8696990013122559, 0.4609334468841553, -0.22397911548614502, -0.619640588760376, 1.4035377502441406, -2.2230279445648193, 0.12000126391649246], [1.1522109508514404, -0.8611095547676086, 0.9602733850479126, 1.3650732040405273, 1.3040423393249512, 0.8874332308769226, -1.221020221710205, 0.8070878982543945, 1.9104762077331543, -0.7691597938537598, -0.9617700576782227, 0.8818909525871277, 0.8165870904922485, -0.24969594180583954, -1.1979176998138428, 0.3536408245563507, -1.5033683776855469, 0.5732463002204895, 0.6221708059310913, -0.3957897126674652, 0.552605152130127, 1.0645514726638794, 0.7015861868858337, -0.8494866490364075, 1.0827354192733765, 0.5288355946540833, -1.2771592140197754, 0.06120702996850014, 0.9908613562583923, 0.15607571601867676, -0.07443705946207047, 0.6070646643638611, 0.44081246852874756, 0.3081631362438202, -0.4201391637325287, -0.560857355594635, -0.26844629645347595, 0.9741589426994324, -0.11158878356218338, -0.21233944594860077, -1.2410873174667358, -2.6974477767944336, 1.4840896129608154, -0.2572776675224304, 0.13119710981845856, -0.9806498289108276, 0.1549709141254425, -0.5639140605926514, -1.206824779510498, 1.8191086053848267, -0.15103919804096222, -0.23536932468414307, 0.6140098571777344, 0.9176124930381775, -0.35147571563720703, -0.6703344583511353, -1.3505818843841553, -0.3908538520336151, 0.7262499928474426, 0.044379230588674545, 0.17936871945858002, 1.741751790046692, 1.3836746215820312, 0.026005735620856285], [-0.7112114429473877, -1.3599467277526855, 0.6700538396835327, -0.28000113368034363, 0.653501033782959, 0.8375175595283508, 0.8408894538879395, -2.2733142375946045, -1.444524884223938, -0.285770446062088, 0.5637919902801514, 0.7741008996963501, 0.7858609557151794, 1.5678067207336426, 0.00011141223512822762, 1.1488723754882812, -0.18117979168891907, 1.7048559188842773, -0.3137725293636322, -1.0673925876617432, -0.5401981472969055, 0.7667896151542664, -0.3129940330982208, 0.6366670727729797, -0.1451941281557083, 0.9784480333328247, 0.08491023629903793, -0.7525590062141418, -0.35604551434516907, -0.21359941363334656, -3.0500783920288086, 0.21516361832618713, -0.5367693901062012, -2.6652820110321045, 1.5562235116958618, -0.4981229603290558, -0.6269843578338623, -1.1867588758468628, 1.0262935161590576, 0.5159804224967957, 0.42324119806289673, -1.0982054471969604, -0.8226912617683411, -1.1646919250488281, -0.41978105902671814, -0.6277528405189514, -0.6845074892044067, -0.5461218953132629, -0.34419411420822144, 1.0296950340270996, 0.20187607407569885, -0.2963458001613617, 0.6393139958381653, 0.7340661883354187, 1.0937596559524536, -0.9529181718826294, 0.09445556253194809, 0.38120681047439575, -0.4951586425304413, 0.023539362475275993, 0.01661268062889576, 0.7388992309570312, 0.1643814891576767, -2.3414595127105713], [-0.13614383339881897, -0.9576860666275024, 0.9232326149940491, 0.21242979168891907, -0.5981715321540833, 0.17823626101016998, 1.0056025981903076, 2.2431485652923584, 0.31617194414138794, -0.004934330936521292, -0.004724842496216297, -0.3520321846008301, 0.48925209045410156, 0.09202505648136139, -2.025296449661255, -1.9560467004776, 0.03975822776556015, 1.6233638525009155, -0.06108362227678299, -1.5048741102218628, -0.8283284306526184, -0.8949213624000549, -1.5882288217544556, 0.5967115163803101, -0.7499154806137085, 0.34458401799201965, -0.9258649945259094, -1.2314128875732422, 1.0895805358886719, -1.3834388256072998, 0.539422869682312, 0.9508789777755737, 0.6004972457885742, 1.6105537414550781, 1.3184298276901245, -1.102294683456421, 0.39000263810157776, 1.206147313117981, -1.7859190702438354, 1.3026174306869507, 0.12679466605186462, 0.655229389667511, -0.5780870318412781, -0.7610928416252136, 1.1621588468551636, 0.9661804437637329, -0.274697482585907, -0.9002560377120972, -1.6216057538986206, -0.18179558217525482, -1.0506012439727783, 0.08208394795656204, -1.3549031019210815, -2.027155637741089, 0.21050937473773956, 0.9308503270149231, -0.6445369124412537, -1.0171513557434082, -1.107764720916748, 0.4631849527359009, 2.1343026161193848, -0.8425592184066772, 0.15241573750972748, -0.7445214986801147], [0.8188328146934509, 0.225943922996521, 0.3372496962547302, 1.13135826587677, -0.5073671340942383, -0.722690761089325, 0.9508028030395508, -0.5137311220169067, -0.27489593625068665, -1.068179965019226, 0.8661752939224243, 1.1878684759140015, -0.36807259917259216, -1.1676955223083496, -0.8252683281898499, -2.72599720954895, -0.282284677028656, -0.6118351221084595, 2.251932382583618, 1.1887034177780151, 0.1953979879617691, 0.8177635669708252, -0.6857269406318665, -2.0296847820281982, -0.4025563597679138, -0.5353153944015503, 2.2628865242004395, -1.4096962213516235, -1.267906904220581, -0.6987038254737854, 2.690647602081299, 0.28227636218070984, 0.04957528039813042, 0.21083490550518036, -1.2376689910888672, -0.28921738266944885, 0.2822485566139221, -1.353005290031433, -1.0272564888000488, 1.2112082242965698, 0.5424755215644836, 1.4446502923965454, -1.6611946821212769, -2.670621633529663, 0.6488019227981567, 0.18083958327770233, 1.2405200004577637, -2.5793840885162354, -0.12597247958183289, -0.2080521583557129, 0.950620710849762, 1.0141761302947998, -2.2314326763153076, 0.9169290065765381, 0.99953693151474, 0.05481226369738579, -1.2168200016021729, 1.7213252782821655, -1.336990237236023, 0.4094688296318054, 0.05454856902360916, 0.7590553760528564, 0.1767847090959549, 1.40961754322052], [-0.3923989236354828, -1.9631948471069336, 1.2292591333389282, -0.4878832995891571, 0.2313160002231598, -0.10475414246320724, 0.2395903766155243, 0.2335529327392578, 1.738388180732727, 1.0161148309707642, -0.401645302772522, -0.5607755184173584, 0.6508249640464783, 1.1892521381378174, 0.48536625504493713, 0.18222576379776, 1.4986056089401245, -0.17873871326446533, 0.5341845750808716, 0.10614003986120224, -0.1158289685845375, 0.2304302155971527, 0.3709735572338104, -0.23931287229061127, 0.8189642429351807, 1.5028681755065918, 1.0726597309112549, -0.7934110760688782, -2.2943811416625977, -1.552259922027588, -1.0792186260223389, -0.4391036629676819, 0.04457741230726242, -0.2986910045146942, 1.9612793922424316, -0.6709873676300049, 0.6442072987556458, 0.4401493966579437, 0.8564370274543762, 2.177133560180664, 0.29770001769065857, 0.243694007396698, 0.7427259087562561, 0.00975593738257885, -0.4153965413570404, -2.286712169647217, 0.44707709550857544, -0.26192060112953186, -0.7106239199638367, 1.007877230644226, -0.8047522902488708, -0.030562464147806168, 0.9936685562133789, -0.07527413964271545, 0.3359176814556122, -0.4234471023082733, 0.2365376502275467, -2.326380968093872, 1.387670636177063, -2.271439790725708, 0.8378665447235107, -1.3757493495941162, -0.8313383460044861, -1.3276365995407104], [-1.9252736568450928, -0.5356155633926392, -1.033920168876648, -0.9681944847106934, -1.2430577278137207, -0.990636944770813, 0.7934362888336182, 0.03347667306661606, -0.0672747865319252, -0.7094284892082214, -0.8173307776451111, 0.6576384902000427, -0.44216182827949524, -1.3514984846115112, -1.5387097597122192, 2.179547071456909, 0.9465739130973816, -1.3544318675994873, 2.230276584625244, 1.5385712385177612, 1.9337213039398193, 0.11987393349409103, -0.7244083285331726, 0.42964234948158264, 0.2509702742099762, -0.24557892978191376, -0.022073091939091682, 0.4200325608253479, 0.11574624478816986, -0.9742551445960999, -0.9000353217124939, -0.49889075756073, -0.23674070835113525, -0.7146692872047424, 0.22876612842082977, -0.09795715659856796, -0.592790424823761, 1.5020726919174194, -1.3582580089569092, -0.6061466932296753, -0.21544092893600464, 0.9654486775398254, -1.0972415208816528, -0.12371493875980377, -0.8119053244590759, 0.5053290128707886, 1.4441189765930176, 0.025011073797941208, -1.0602645874023438, -0.3616403639316559, -0.6434251070022583, -1.0391286611557007, -0.4349401295185089, -1.010300636291504, 0.8185170888900757, -0.27180275321006775, -0.48803433775901794, 1.0642939805984497, 0.4031182825565338, 0.04832176864147186, -1.2988847494125366, -0.6753779649734497, 0.7016761898994446, 1.1036388874053955], [-2.59624981880188, -0.4341534376144409, -0.6474883556365967, -0.8410875797271729, 0.20957648754119873, 0.8938086032867432, -1.3961503505706787, -0.166518896818161, -0.4529419541358948, -0.7606637477874756, -1.4473226070404053, 0.28621700406074524, 0.5469644665718079, 0.00550966989248991, 0.07791285216808319, 0.8050181269645691, 0.5113117098808289, -1.1430479288101196, 0.8730564117431641, -0.4249444305896759, -1.4982218742370605, 0.4763994514942169, 1.4962348937988281, 1.743571162223816, -0.3467932939529419, 0.40265971422195435, -1.5193777084350586, 1.3589155673980713, 1.3518122434616089, -0.299943208694458, 0.3292958736419678, 0.6811888813972473, 1.4795430898666382, -0.3894768953323364, 1.7496469020843506, 0.2432374507188797, 0.8873662352561951, 0.7830146551132202, 2.1393721103668213, -0.77956622838974, 1.0693258047103882, -1.3710360527038574, 1.5440294742584229, -1.3213919401168823, -0.37647369503974915, 0.8423563241958618, 0.32252684235572815, 0.20736905932426453, -0.24522900581359863, -1.3342792987823486, -0.2858847379684448, -1.1683365106582642, 1.8820579051971436, 1.129295825958252, 0.7482503652572632, 1.0262678861618042, 1.1947892904281616, -1.321506381034851, 0.6129544377326965, -1.9918493032455444, 0.45622718334198, -1.4778224229812622, -0.5014967918395996, -0.0689871609210968], [1.196150541305542, 0.8931716084480286, -0.8284577131271362, -0.6568727493286133, 0.24851340055465698, 1.960501790046692, 0.8163495659828186, -0.6061748266220093, -0.7548922300338745, 1.2446839809417725, -1.0072417259216309, 0.3858626186847687, 0.25976401567459106, 0.12329568713903427, 0.7056897878646851, -0.7235349416732788, -2.4150476455688477, -1.2561564445495605, -0.1388818770647049, -2.4524459838867188, -0.06923641264438629, -1.689549207687378, -1.9284476041793823, 0.6561532616615295, -2.309248447418213, 0.022332940250635147, 1.4443167448043823, 0.8576898574829102, 0.7214751839637756, 0.9694739580154419, 1.0444393157958984, 0.11780081689357758, 1.5595757961273193, -1.8960295915603638, 1.6468347311019897, -0.291254460811615, -1.610574722290039, -1.10293710231781, 0.7213839292526245, 0.5946252346038818, -0.3155459761619568, -1.8266749382019043, 0.875214695930481, 0.19991405308246613, 0.7166163325309753, -0.9894196391105652, 0.8083374500274658, 1.044446587562561, -0.00685737794265151, 0.3778553307056427, -0.641068696975708, -1.1732206344604492, 0.6415321826934814, -0.08196983486413956, -0.422127366065979, 1.3593027591705322, 0.5928869843482971, -1.0309081077575684, -0.6337383985519409, 0.12839621305465698, 1.0749858617782593, -0.606185257434845, 0.4191269874572754, -1.6653828620910645], [-0.9855674505233765, -2.067826986312866, -0.21175113320350647, 0.42344263195991516, 0.2521679997444153, 0.5809203386306763, 1.173254370689392, 0.13270601630210876, -1.302862524986267, -0.27714022994041443, 0.5033324956893921, -0.10256900638341904, -0.3453995883464813, -0.49560630321502686, -0.021704990416765213, 2.029937982559204, -2.3451576232910156, 2.3157763481140137, -1.050079345703125, 0.8462366461753845, -0.2846859097480774, -0.6548152565956116, -0.08655306696891785, -0.8332651257514954, 0.7749155759811401, -0.36356624960899353, 1.0054330825805664, -1.3480725288391113, 0.5396520495414734, -0.09201186150312424, 0.006354151759296656, -1.5078893899917603, 0.7805303931236267, -0.6798698902130127, 0.027873480692505836, -1.0795270204544067, 0.6187450885772705, -0.13647815585136414, -1.2971428632736206, 0.12373103201389313, -1.072911024093628, 1.7701396942138672, -0.21024952828884125, -1.3786991834640503, 0.8601099252700806, 0.027986139059066772, 1.3674206733703613, -0.03295120596885681, -0.4988812804222107, -0.03415370732545853, 0.03301643207669258, -1.467294454574585, -0.8041825294494629, -0.10544302314519882, 1.006542444229126, -0.9071381688117981, -0.030635500326752663, 0.41476958990097046, 0.2926490604877472, 1.021991491317749, -0.2213682234287262, -0.2896308898925781, 0.05206318199634552, -0.17201071977615356], [-1.2828686237335205, -0.36176520586013794, 0.7493061423301697, 0.49255576729774475, -1.4480533599853516, -1.0260599851608276, 1.684624195098877, -2.0216054916381836, 0.2880081832408905, 0.3936597406864166, -0.876722514629364, 0.11249949038028717, 0.09070389717817307, -0.715788722038269, -2.872056245803833, 0.019049573689699173, -0.13598309457302094, 0.06521379202604294, -1.9889380931854248, -0.17208868265151978, -1.2321653366088867, -0.19099029898643494, 0.9699150323867798, -0.13684441149234772, 0.9900851249694824, 0.6044243574142456, -0.9367823600769043, 0.08092683553695679, -0.4760706424713135, -1.3653496503829956, 1.0392305850982666, 0.01978985406458378, -0.6800455451011658, 0.3268403112888336, -0.46892815828323364, -2.021465301513672, -3.410759449005127, 0.002405952662229538, -0.4234403669834137, 1.435890793800354, 2.2500386238098145, -0.0939718708395958, -0.8153442144393921, -0.4032563865184784, -1.2701551914215088, 0.3791835308074951, 0.777477502822876, -0.475077748298645, -0.6857598423957825, -0.36673128604888916, 0.41479119658470154, 0.17146320641040802, -1.4026743173599243, -0.7147824764251709, -1.06709623336792, -0.8938570618629456, 0.7463391423225403, -0.0038826283998787403, -0.2062915563583374, -0.5838021039962769, -0.22142727673053741, 0.8801996111869812, -0.5567657351493835, 2.094883680343628], [-1.9473283290863037, 0.8839211463928223, -0.22692547738552094, -2.6309337615966797, -0.801217257976532, 1.4285533428192139, -0.9663419127464294, 0.3703294098377228, -0.0848514512181282, 0.5345991849899292, -0.7571887373924255, 0.5068855881690979, 0.7365870475769043, 0.6753508448600769, 1.236938238143921, -0.28251171112060547, 1.0004085302352905, -0.4840908944606781, -1.6457748413085938, -0.9121395945549011, 0.7400797605514526, 1.3819245100021362, 0.7245270013809204, 1.0932397842407227, -0.5694173574447632, -0.3839174211025238, -0.7337916493415833, 0.9559639096260071, -1.6495630741119385, 1.0906622409820557, 0.15720392763614655, 0.3833081126213074, -1.8740293979644775, -0.03154440224170685, -0.8539875745773315, 0.18716119229793549, 0.5738545060157776, 1.0650299787521362, 0.2024342119693756, 0.29898932576179504, -0.6824160218238831, 0.5259116888046265, 1.2606821060180664, 0.5969663262367249, 0.6623898148536682, -0.548983097076416, 0.8285694122314453, -0.866033673286438, 0.5179826617240906, -0.5712169408798218, -0.5918384790420532, -2.5136830806732178, 1.3734369277954102, 0.8647612929344177, 1.1626673936843872, -0.02133990079164505, 0.36646631360054016, 1.3329310417175293, 0.4851802885532379, 0.7074896693229675, 1.695556402206421, -0.40719717741012573, 0.9424090385437012, -0.41466963291168213], [1.1460658311843872, -0.7652804851531982, -0.10337665677070618, 0.9294406771659851, -0.1888469159603119, 0.10789879411458969, 0.1411595493555069, -0.488505095243454, 0.7062004804611206, -0.8581461310386658, -0.34412381052970886, 0.49328768253326416, 0.17867404222488403, -0.41782769560813904, 1.395212173461914, -0.6224702000617981, -0.43515780568122864, 1.2975069284439087, 1.7969465255737305, -1.376882791519165, -0.06948529183864594, 0.09486710280179977, 1.7893577814102173, 0.7403591871261597, 0.10682229697704315, -0.4908585548400879, -0.24471284449100494, -1.2191840410232544, -1.8970692157745361, -2.1385154724121094, 1.0438752174377441, -2.117418050765991, -0.3458040952682495, 0.346252977848053, -0.4831644892692566, -0.4111417233943939, -0.9333957433700562, 0.46743282675743103, -0.2320849895477295, -1.1324589252471924, -0.8715230226516724, -0.13419246673583984, -0.24634763598442078, 0.830362856388092, 1.278818130493164, 0.3991469740867615, 1.4321551322937012, -0.20766125619411469, 0.12226174026727676, -0.29347288608551025, 0.09768103808164597, 0.6044914126396179, 0.9120675921440125, 1.0873295068740845, -1.549303412437439, -1.5439096689224243, -0.42225533723831177, 0.08730673789978027, -0.42384645342826843, 1.1665232181549072, -1.1394295692443848, 0.4231100082397461, 0.9916535019874573, 1.1826962232589722], [-1.1392706632614136, 0.017726341262459755, 0.3246687948703766, -0.8285526633262634, 0.1592799872159958, -0.24204450845718384, 0.35057178139686584, -0.17861883342266083, -1.60305655002594, -0.986258327960968, 0.15426912903785706, 1.0597929954528809, 0.2559460997581482, -0.5757473707199097, 0.432146281003952, -2.9091854095458984, -1.6889206171035767, -1.032545804977417, -1.2372335195541382, -0.5439232587814331, 0.8411709070205688, 0.9757165312767029, -0.8640397787094116, 0.0013574030017480254, -1.8317601680755615, 0.4697156548500061, -0.3585715591907501, 0.2102869153022766, 0.27693474292755127, 0.18131212890148163, 0.10504976660013199, 1.428298830986023, -0.5922436714172363, -0.004398747347295284, 0.9171425700187683, -0.2673644721508026, -1.2753965854644775, -0.5882272124290466, 1.3740586042404175, 0.6629738211631775, 0.9681528806686401, -0.4150577783584595, 0.4172709584236145, -0.7688656449317932, -1.6192256212234497, -0.6233174800872803, -0.09782043844461441, 0.3680165112018585, -0.7295221090316772, 0.29073020815849304, 0.7819344401359558, -0.697258472442627, 0.10761896520853043, 0.2256477028131485, 0.6052192449569702, -0.7873952984809875, 1.4314212799072266, 1.0640182495117188, 0.19199559092521667, 0.1789989471435547, 0.824044942855835, 0.8336897492408752, 0.07289151102304459, 0.6935566067695618], [0.7797342538833618, -1.168531060218811, 0.31532907485961914, -1.0146265029907227, 1.1638528108596802, -2.5917680263519287, -1.6870094537734985, -0.28545787930488586, 1.175657033920288, -0.4607531428337097, -0.39430609345436096, 1.0460411310195923, 0.2517124116420746, -0.5533002018928528, -1.375571370124817, -0.6404698491096497, 1.3398369550704956, -0.963391900062561, 0.4837137460708618, 1.2748818397521973, 0.054556068032979965, 0.9111636281013489, -2.0859813690185547, -0.5607131123542786, -2.297396183013916, -1.4180437326431274, -0.6354114413261414, -1.0123584270477295, 0.6313403248786926, -0.3783622086048126, 0.49583396315574646, -2.1751151084899902, 0.34060248732566833, -0.9940697550773621, -1.0006012916564941, 0.6632673144340515, 0.06733135133981705, -1.0793325901031494, -1.278277039527893, 1.3160876035690308, 1.8185163736343384, -1.5607764720916748, 1.2005863189697266, -1.1583775281906128, 0.09962380677461624, -0.7467050552368164, 0.050864141434431076, 1.7578885555267334, -1.0082343816757202, 0.29043155908584595, -0.031196407973766327, 0.21835413575172424, -0.2551383674144745, -0.36623525619506836, 0.36947572231292725, 0.37294861674308777, 0.9037877321243286, -0.9606621861457825, -0.39228203892707825, 0.6744313836097717, -0.13381153345108032, -0.9411712884902954, -2.2019591331481934, -1.4121214151382446], [1.1952663660049438, -0.9531920552253723, -0.5678755640983582, 2.024693012237549, 0.6116699576377869, 0.7416099905967712, 0.8124136328697205, -0.5226707458496094, -0.347590833902359, 0.15667225420475006, 1.4147062301635742, 0.3594307601451874, 0.07372228801250458, 0.35116857290267944, 1.8675715923309326, -0.855288565158844, -1.5274896621704102, -2.0358505249023438, -1.469058632850647, 0.3602774441242218, 0.811252236366272, -0.37815120816230774, 0.7859379649162292, 0.416263610124588, 0.3004525303840637, -0.26065221428871155, 1.0270613431930542, 0.49193117022514343, -0.0010312972590327263, 0.43306881189346313, 0.194182887673378, -0.5680527091026306, -1.03594970703125, -1.332279920578003, -1.657525897026062, 1.7145156860351562, 0.4819204807281494, -2.107146978378296, -1.0273770093917847, 0.7905216217041016, -0.6410865187644958, -0.18834948539733887, -0.21009938418865204, 0.2589930593967438, -0.32470953464508057, -0.968748152256012, 1.843100666999817, -0.14011161029338837, 0.5134006142616272, -0.6460631489753723, -0.6660273671150208, -0.5288572907447815, 0.8969007730484009, -0.8467538356781006, -0.5779960751533508, -1.8515771627426147, 1.12239408493042, 1.4615709781646729, 1.324204444885254, -1.2254537343978882, 2.008528470993042, -1.3772690296173096, -0.3046781122684479, 0.11378005892038345], [-0.37193822860717773, 0.29093530774116516, -0.43912237882614136, 0.20789168775081635, -1.3345317840576172, -0.289825439453125, 0.7155317664146423, -1.3326760530471802, -1.4550886154174805, 2.0330970287323, -2.3892688751220703, -0.30449697375297546, 0.13950012624263763, 2.563516855239868, -0.03673363849520683, 1.4733105897903442, 0.7627136707305908, 0.4505525827407837, 1.3410838842391968, 0.2821744382381439, -1.7938381433486938, 0.8132573366165161, 1.6994853019714355, 0.5107231736183167, -0.3835281431674957, 1.1891717910766602, -0.29061368107795715, -0.040408674627542496, 0.31400248408317566, 0.870876669883728, 0.25432780385017395, 0.905754804611206, -0.15126335620880127, 1.4943516254425049, -1.1285053491592407, -0.6010909080505371, -0.4685159921646118, -0.32070690393447876, -0.672977864742279, 0.5583544969558716, 0.058086372911930084, 0.7057058811187744, -0.19399414956569672, -1.1092164516448975, -0.2555115222930908, -0.1696893274784088, 0.09545149654150009, -1.3537402153015137, 0.5417948365211487, -0.596811830997467, 0.39545801281929016, 0.3901413381099701, 0.8009989261627197, 0.2619484066963196, -0.8395996689796448, -0.4123150110244751, -0.5348933935165405, 1.201875925064087, -1.0127090215682983, 2.2449495792388916, 1.0623198747634888, -1.5850005149841309, 0.35010582208633423, 1.468514084815979], [1.0755401849746704, -0.866460919380188, -0.8066248893737793, 0.12144475430250168, -1.527189016342163, -0.3417748510837555, 1.4252691268920898, -0.045091740787029266, 2.1403393745422363, -0.4356544315814972, -2.345341205596924, 0.13386888802051544, -0.3933102488517761, 1.4267065525054932, -1.0648998022079468, 0.38382312655448914, 1.980928897857666, 1.233102560043335, 0.9550763368606567, -0.45934656262397766, -1.7322319746017456, 0.517710268497467, 1.425826907157898, 0.0323026068508625, 0.4828492999076843, -1.42383873462677, -1.7222869396209717, 0.42800939083099365, -0.682459831237793, 1.037099838256836, -2.69728422164917, 0.28906089067459106, 0.3915124237537384, -0.7237711548805237, 0.18518805503845215, 1.654887318611145, -1.0366649627685547, 0.6878035068511963, -0.9513461589813232, -0.21964354813098907, 1.0821090936660767, 1.1406524181365967, -1.5571508407592773, 0.9653688669204712, -0.43326765298843384, -1.272013783454895, 0.7631168961524963, -1.7963893413543701, 2.2130730152130127, 1.5189528465270996, -0.9924381971359253, 0.4045315682888031, 0.2637641727924347, -0.8636890053749084, -1.8413195610046387, 0.1074567586183548, 1.0141884088516235, -2.296952486038208, -0.6842489838600159, 0.5279267430305481, 1.8555458784103394, -2.3954501152038574, -1.0238606929779053, -0.05270610377192497], [-1.9264134168624878, 0.1878359615802765, 0.024782106280326843, -0.5859703421592712, 0.9426833391189575, 1.2871506214141846, -2.708728313446045, -1.1387715339660645, 1.1113024950027466, 1.3944323062896729, -1.1060317754745483, 0.5211117267608643, -1.1074564456939697, 0.4161727726459503, -0.04014817997813225, -1.1387518644332886, -0.11245761811733246, 0.40395721793174744, 1.103822946548462, 1.7742685079574585, 1.1026993989944458, -0.24282807111740112, -1.01426100730896, 1.7614887952804565, -0.03534630686044693, 1.7800506353378296, -1.4469496011734009, -1.4839003086090088, -0.4485132396221161, -0.017443839460611343, -0.29625028371810913, -0.6669216752052307, -0.16867876052856445, 0.300801545381546, -0.9320660829544067, 0.19118446111679077, 0.2455211877822876, 0.07619685679674149, -1.7444946765899658, -2.3130362033843994, -0.5940293073654175, -0.3743908703327179, 0.870963990688324, 0.15606044232845306, -0.6439202427864075, -1.0261415243148804, 0.31016454100608826, 2.084235668182373, -0.20296719670295715, 0.9744802117347717, 0.4714324474334717, 0.07880176603794098, -2.2567555904388428, -0.5544873476028442, -1.7399085760116577, 1.057785153388977, 0.8466775417327881, 1.5352144241333008, -0.11326280236244202, -0.6522232294082642, -0.7120614051818848, 0.48762911558151245, -2.083610773086548, -1.0689136981964111], [-0.4504289925098419, 0.7606430649757385, 0.7909485101699829, 0.3451993763446808, -0.28545162081718445, -0.683117687702179, 0.6641706228256226, -0.08048313856124878, 0.8967756628990173, 0.7367560267448425, 1.5482255220413208, -1.309667706489563, -0.1407058984041214, -1.1274994611740112, -0.7468978762626648, -0.283085435628891, -1.2371623516082764, -0.3206806480884552, 0.5624563694000244, -0.46719908714294434, 0.2144704908132553, -1.5721783638000488, -1.6523123979568481, -0.5150272250175476, 0.6885939836502075, 1.0827795267105103, 1.8749531507492065, -1.392075777053833, 1.06583571434021, -1.0080112218856812, 0.5797934532165527, 0.5400465726852417, 0.4676903486251831, 1.1965500116348267, 1.279741883277893, -2.3671300411224365, 1.5430066585540771, 0.7478662729263306, 0.3427899181842804, 1.2158501148223877, -0.05298012122511864, -0.37109375, 0.09147696942090988, 0.0004344474000390619, 0.15892313420772552, -0.4395933449268341, 1.1295231580734253, 0.9170593023300171, -0.8689737915992737, 0.04975786432623863, 0.6981801986694336, 1.0491732358932495, -1.0538904666900635, 0.9663045406341553, 0.02815321832895279, -1.7963446378707886, 2.740482807159424, -0.8125145435333252, -0.4703967869281769, -0.7722104787826538, 1.0847400426864624, -0.44509223103523254, 0.8891341090202332, 0.08613049983978271], [-0.1446256935596466, -0.18023119866847992, 0.7532961964607239, -0.030698716640472412, 0.6379543542861938, 0.044318683445453644, -0.6895556449890137, -0.8560627698898315, -0.08478345721960068, 1.771276831626892, -0.1589081883430481, 0.19468064606189728, -1.077915072441101, 0.05584418773651123, -0.35459935665130615, -1.3235915899276733, -1.6388587951660156, -1.1263092756271362, 0.4374649226665497, 0.7892702221870422, -0.7173119783401489, -0.06825960427522659, 0.458507776260376, -1.000563383102417, 0.5648830533027649, 0.6257228851318359, 0.2203211486339569, 1.2752039432525635, -0.7118791937828064, 1.4120666980743408, -0.3926064372062683, 0.10533779114484787, -2.009765386581421, 0.9784575700759888, -1.9556779861450195, -0.2539611756801605, 1.351868987083435, -0.1785421073436737, 0.1474587768316269, -0.07505600154399872, -0.9125485420227051, 1.3707263469696045, -1.0012810230255127, 0.8863283395767212, -0.8603987097740173, 1.1324231624603271, -0.1848578155040741, -0.8516769409179688, 0.3448469638824463, 0.001544905942864716, 1.5082887411117554, -1.5552490949630737, 0.3557128608226776, 1.3496006727218628, -0.7844301462173462, 0.781508207321167, 0.9123189449310303, 0.37957969307899475, 1.4159257411956787, -1.1821434497833252, 0.387717604637146, 0.9594094753265381, -0.7531696557998657, -0.11828292906284332], [0.10590197145938873, 0.30556970834732056, -2.9551682472229004, 1.046492099761963, -0.9449437856674194, 0.49655741453170776, -0.32438936829566956, 0.40485888719558716, -1.6038156747817993, -0.2175317108631134, 0.381512314081192, 0.41189253330230713, 0.7192922830581665, -0.6966369152069092, 3.020876169204712, -0.6807704567909241, 1.0016913414001465, 0.855298638343811, 1.009738564491272, -0.3266168236732483, -1.6783862113952637, 2.0248494148254395, 0.1806393563747406, 1.8951373100280762, -0.9042435884475708, -1.2660999298095703, 0.07131377607584, 1.3584798574447632, 0.3204064965248108, 0.3858242928981781, -0.31789371371269226, -0.0025947741232812405, -0.37800392508506775, -0.0156293623149395, 0.9964139461517334, -1.4984115362167358, -0.27042558789253235, 0.4246867001056671, -0.1958283632993698, -0.06938823312520981, 1.3565938472747803, -1.9732528924942017, -1.1548755168914795, 0.20634691417217255, -0.7359707355499268, -0.9463225603103638, -1.5221954584121704, -1.1986662149429321, 1.5649189949035645, -0.9707534909248352, -0.4421177804470062, -0.569047212600708, -0.2694132626056671, 0.8802409768104553, 0.7093548774719238, -0.30036184191703796, 0.30882880091667175, 0.9562672972679138, -1.6531405448913574, 0.8010122776031494, 0.5904925465583801, 1.1199010610580444, -1.4124614000320435, -0.9514600038528442], [0.4686441421508789, 0.3383786678314209, -0.22807128727436066, 0.6234761476516724, 0.6723279356956482, -1.5502538681030273, -0.953707218170166, -0.8510483503341675, 0.34997814893722534, 0.3790243864059448, -0.5795658230781555, 0.6554597616195679, -0.13119915127754211, -0.8063527345657349, 0.5713778138160706, -1.1170158386230469, -0.1616954207420349, 1.0547045469284058, 0.1551830917596817, 0.8167686462402344, 1.1569002866744995, 0.15699179470539093, -0.031839486211538315, -0.37492501735687256, 0.06071483716368675, 0.18662932515144348, -1.1645450592041016, -0.3680782914161682, 0.9731770157814026, 0.7577802538871765, 1.9189938306808472, 0.32438924908638, -0.624421238899231, -0.44811201095581055, -0.3524586260318756, 1.8567525148391724, -0.504043459892273, 1.0936928987503052, 1.5233920812606812, -1.350260615348816, -1.166566252708435, 2.267537832260132, -1.4590169191360474, -1.2982277870178223, 0.2920636534690857, 1.68783700466156, -2.142129421234131, 0.7795841693878174, -0.9015547037124634, 0.3746525049209595, 0.4668670892715454, -0.21557022631168365, -1.3233907222747803, -1.567217230796814, 0.6442149877548218, 2.0142321586608887, 1.0907342433929443, -1.8322243690490723, -0.8622822761535645, -0.8291427493095398, -0.22954261302947998, -1.4703444242477417, -0.5870670080184937, -0.4791465401649475], [-0.17232297360897064, 1.3747402429580688, -0.2444629967212677, 0.6403143405914307, -0.3559778928756714, 0.9968836307525635, 0.4646932780742645, -1.6241586208343506, 1.1780070066452026, -2.690777540206909, -0.13839514553546906, -0.5102086067199707, -0.8829164505004883, -0.3634292185306549, 0.298702210187912, 1.2740412950515747, -0.5790541172027588, 0.6584975123405457, -1.2192515134811401, 0.46644526720046997, 0.4136117696762085, 0.47179093956947327, -2.0590736865997314, -0.302132248878479, -0.30191677808761597, -0.4265250861644745, 0.5058340430259705, 1.3102554082870483, 0.5958877801895142, 0.1630876660346985, 1.0922276973724365, 0.5326712131500244, 1.449535846710205, 0.012659606523811817, 2.1413755416870117, 1.5056663751602173, -0.5455670356750488, -0.4929395616054535, 0.10416997969150543, -0.07620653510093689, 0.1518033891916275, -1.9373043775558472, 0.39350825548171997, -0.0650724396109581, -1.6246144771575928, 0.17236831784248352, -0.5354215502738953, 1.7134050130844116, -0.8800879120826721, 1.0904181003570557, 0.609486997127533, -0.5968488454818726, 1.1279122829437256, 1.0233123302459717, -1.8082013130187988, 1.416710376739502, 0.13269874453544617, 0.4094776511192322, -0.262912780046463, 0.2774140536785126, -0.5223369002342224, 0.3938618302345276, 1.6618059873580933, -1.1521443128585815], [-1.77305006980896, -0.13071314990520477, -0.28054168820381165, 0.6303998231887817, 2.3858537673950195, -0.5512115955352783, -0.9031360149383545, 1.9653466939926147, -0.06916159391403198, 0.34730294346809387, 2.5083720684051514, -0.9320767521858215, -0.38338884711265564, 0.660925567150116, 0.24508841335773468, 2.0687570571899414, 0.19707106053829193, 1.6334915161132812, -0.22917982935905457, -0.2973264753818512, -0.48952123522758484, -0.8965811133384705, 0.031792089343070984, -0.8659391403198242, 0.04969308152794838, -1.0807229280471802, 0.13776592910289764, -0.01699916459619999, -0.15880119800567627, -0.6932259798049927, 2.3612802028656006, 0.8205217123031616, -0.9996828436851501, 0.47064465284347534, 0.49061864614486694, 1.8254448175430298, -0.9659821391105652, -1.6403111219406128, 0.009099865332245827, 0.20307621359825134, -1.2885746955871582, -0.5132784843444824, -0.9249241352081299, 1.5487815141677856, 1.6498595476150513, 0.045407891273498535, 0.7787957191467285, -0.5733895301818848, 1.8940279483795166, 0.17475734651088715, 0.6643383502960205, 0.48136094212532043, -0.375593364238739, 0.12985575199127197, 1.65351140499115, -1.0926239490509033, -1.6039899587631226, -0.5945323705673218, -0.031856294721364975, 0.7875966429710388, 0.8683145642280579, 1.675704836845398, 1.252091407775879, -1.0883772373199463], [-1.072230577468872, -0.7038165330886841, 0.778654158115387, -0.23922473192214966, -0.8986948132514954, -0.3535000681877136, -1.0948419570922852, -0.9418126344680786, 2.130903720855713, 1.1963069438934326, 1.4206351041793823, 0.918113112449646, -0.4514179229736328, -0.22004634141921997, -0.07325998693704605, 0.5100481510162354, -1.589571475982666, -0.49615564942359924, -1.0989614725112915, 0.8897954225540161, 0.08079379051923752, 0.3613390326499939, -0.2989387810230255, 1.5914877653121948, 0.8030136227607727, -0.9197304844856262, 0.4893334209918976, -0.9619457721710205, -0.3588148057460785, -1.511396884918213, -0.9724870920181274, -0.33495792746543884, 0.6751991510391235, 0.001710027689114213, -0.38645297288894653, -0.8576295375823975, 0.678436279296875, 0.35597631335258484, 0.42907553911209106, -0.21743057668209076, -0.6143850684165955, 0.09159160405397415, -0.13861732184886932, -1.4146625995635986, -1.7398840188980103, -0.3814912438392639, 0.4370872974395752, -0.41646209359169006, 0.15852205455303192, 0.33228856325149536, 0.01098412275314331, 1.812061071395874, 1.172806978225708, 0.5158509612083435, -0.41092270612716675, -0.11783953756093979, -0.057347238063812256, 1.0877732038497925, -0.6244168877601624, 0.9815433025360107, -0.9080823659896851, -0.9515701532363892, -1.0726492404937744, 0.6745947599411011], [-0.7005287408828735, -0.7901966571807861, 1.1309151649475098, -0.7279446721076965, 1.247294545173645, 0.6798029541969299, 0.3389533758163452, -1.5571305751800537, -1.9544670581817627, 1.4534703493118286, -1.0030555725097656, -1.1306089162826538, -0.7060874104499817, 0.2779756486415863, 0.7950787544250488, 1.1754393577575684, 1.8523941040039062, 0.9429688453674316, -1.1623295545578003, 0.40721985697746277, -0.18571874499320984, 0.008882954716682434, -0.4549761712551117, 1.820593237876892, 0.9968898296356201, -0.017894407734274864, -2.397493362426758, 0.7043912410736084, -0.7549939751625061, -1.5567727088928223, 2.8469367027282715, 0.12319924682378769, 0.5271539092063904, 1.346110224723816, -0.9447208642959595, -1.0766021013259888, -1.6889865398406982, -0.24046233296394348, 0.6323378086090088, 0.5098248720169067, -0.4966486692428589, 0.8262507319450378, -0.22519107162952423, 0.4471920430660248, -0.7572899460792542, 0.0699789747595787, 0.604002833366394, -0.6000616550445557, -0.5619892477989197, -1.7887086868286133, 0.746781051158905, -0.5015547275543213, 0.13208480179309845, 0.9431244134902954, 1.1431516408920288, 0.23999889194965363, -0.0759352594614029, -0.9771127700805664, 0.01648586057126522, 0.6241045594215393, 0.9470475316047668, 0.8195294737815857, -0.3075895607471466, 0.7018835544586182], [-1.5776195526123047, -0.15222615003585815, -0.6612546443939209, 1.25253164768219, -0.031062966212630272, 1.9101279973983765, 0.22317178547382355, -0.2742498815059662, -1.5376917123794556, 0.309577077627182, 1.415150761604309, 0.5931277871131897, 0.13060817122459412, 0.7530291080474854, -2.328233480453491, -0.28525421023368835, 1.5955684185028076, 0.547447681427002, -0.29127898812294006, 0.49421238899230957, -0.2442912459373474, 0.2552768588066101, 1.416142463684082, 1.622369647026062, -0.0284016951918602, -0.05296797677874565, 0.6768034100532532, 1.140950083732605, 0.24779962003231049, 1.2265890836715698, -0.3707411587238312, -0.7867146730422974, 0.8545647263526917, 2.03775691986084, 0.504213809967041, -0.2581934928894043, 0.8119981288909912, -1.155842661857605, -1.3254966735839844, -0.20532631874084473, 1.5534651279449463, 1.7147483825683594, -0.775439977645874, 0.9689045548439026, 0.30462920665740967, 0.23364917933940887, -0.6003646850585938, -0.7764399647712708, 0.20209404826164246, 0.8565536737442017, 0.5163028240203857, -0.9818748235702515, 0.6437293887138367, -0.4107506275177002, 0.5602909326553345, 0.4957481920719147, 0.0056379386223852634, 1.4870895147323608, 1.9162168502807617, 0.817704439163208, -1.5793060064315796, -1.2356516122817993, -0.14170551300048828, -0.0032875570468604565], [0.6225981712341309, -1.015338659286499, 0.8210765719413757, 0.8863488435745239, 0.8583837151527405, 1.2629364728927612, 0.6608034372329712, -0.32475754618644714, 0.12143079936504364, 0.9860590696334839, -0.8292254209518433, -0.20720750093460083, 0.3587811589241028, 0.9194917678833008, -0.661011278629303, 0.4405365586280823, -1.2419955730438232, -0.8708215355873108, 1.1320116519927979, -0.33936360478401184, 1.4289017915725708, 0.10166759788990021, -0.9592081308364868, -1.159518837928772, 0.812555730342865, -0.9865497946739197, 1.0805402994155884, 0.7708603739738464, 0.7641809582710266, -3.318676710128784, 0.18927539885044098, -1.166042447090149, -2.02060866355896, 2.3723058700561523, -0.33383265137672424, -0.6574288606643677, 0.6453055143356323, -0.3535136878490448, -1.5785493850708008, 0.44061902165412903, 0.526453971862793, -0.10512223839759827, 0.04149068519473076, 0.5847721099853516, -0.43665462732315063, 0.15825356543064117, -0.4268261790275574, -1.0412464141845703, 0.009753813035786152, -0.33651241660118103, -0.7607517242431641, 1.952008843421936, -0.43508991599082947, -0.314692884683609, 1.5724389553070068, -1.5118399858474731, -1.2836203575134277, -0.6496589183807373, 0.5868014693260193, -0.37462106347084045, 0.2220037877559662, -0.612392008304596, 0.144257053732872, -0.3020739257335663], [-0.006201920565217733, -0.5813166499137878, 0.2230401635169983, 0.975496768951416, -0.9783262610435486, 0.41860026121139526, -0.33867010474205017, 0.6605566143989563, -0.15805202722549438, 0.1891656517982483, -0.6708555221557617, -1.1205426454544067, -1.2387746572494507, -1.3163260221481323, 0.6374762058258057, 0.5987692475318909, 0.3891986012458801, -0.5888651013374329, 0.811627209186554, 1.6492457389831543, 0.6122420430183411, -1.5460094213485718, -1.250999093055725, 0.6028200387954712, 0.6280975341796875, 0.918705940246582, 0.9824479818344116, 1.5282714366912842, -1.0275194644927979, 0.06706592440605164, 2.107052803039551, 0.03558710217475891, -1.05012047290802, 0.769930362701416, 0.7337340116500854, 0.08401068300008774, 0.3487522602081299, -0.8515830039978027, -0.879439115524292, 1.63765549659729, -1.8143163919448853, -0.36345529556274414, 0.4704659581184387, -0.392717570066452, 0.34495165944099426, -1.0753498077392578, -0.2665026783943176, 0.3489387333393097, -0.3988332748413086, 1.977439284324646, 0.3405036926269531, -0.6572275757789612, -0.608590304851532, 0.5756403207778931, 0.052920084446668625, -0.8141191601753235, 0.1722760945558548, 1.38326895236969, -0.2855273187160492, 1.492378830909729, 0.1659373939037323, -0.9074667692184448, 0.40997615456581116, 0.3393026292324066], [0.4938828945159912, -0.771282970905304, -0.5693448185920715, 0.05486772581934929, -0.14069530367851257, -0.6048325300216675, -1.3087610006332397, 0.07073158770799637, -1.177369475364685, 1.0903370380401611, 0.439748078584671, -0.743994951248169, 1.3887057304382324, 1.18380606174469, 0.7596907615661621, -1.0497013330459595, -0.910147488117218, 1.7125598192214966, -0.32714247703552246, 1.7823716402053833, 0.3671191930770874, -0.3406718671321869, -2.25907301902771, 0.788252055644989, 0.79499751329422, 0.7590463161468506, 0.24124659597873688, 1.5438708066940308, -0.7455495595932007, 0.1455237865447998, -0.22572220861911774, -1.3809547424316406, 0.968982458114624, 0.558674156665802, 0.024082496762275696, 0.7041558027267456, 0.5409566760063171, 0.3957552909851074, 1.4810467958450317, -0.41974812746047974, 1.3467762470245361, -1.7280627489089966, -1.6200538873672485, 1.2933399677276611, 0.17252689599990845, -0.10411427915096283, 1.5792204141616821, 0.7833082675933838, 0.10115328431129456, -0.20795392990112305, -0.6432222127914429, 0.9434463977813721, 0.36458057165145874, -0.42750537395477295, 0.6648150682449341, 1.1889773607254028, 0.6751224994659424, -0.3940891623497009, -0.8137531280517578, 1.3754799365997314, 0.12402001768350601, -0.46862539649009705, -1.7795039415359497, -1.7710552215576172], [-0.3231590688228607, -0.16767922043800354, 1.2665839195251465, 0.5362247228622437, -0.08541466295719147, 0.22126512229442596, -0.9695811867713928, -1.0303095579147339, 0.6539689898490906, 0.48969143629074097, 1.0770304203033447, -0.6417527198791504, 0.36128005385398865, 1.5334757566452026, -0.27793899178504944, -2.3700010776519775, 1.2756075859069824, -1.6259028911590576, 0.1187405064702034, -0.1187298521399498, -0.06265329569578171, -0.6957210898399353, -0.47606953978538513, -0.2872037887573242, 0.11423926800489426, 0.08801203221082687, 0.3946431875228882, 0.45601513981819153, 1.296679139137268, -1.4607399702072144, 0.5860612988471985, 0.9790870547294617, 1.6880457401275635, 0.30639469623565674, 0.8473414182662964, 0.3933004140853882, -0.08946207910776138, 0.3810328245162964, -1.2133934497833252, -1.4995118379592896, -0.3638799786567688, 0.5986877679824829, -1.111840009689331, 2.475172519683838, -0.5455747842788696, -0.3529059588909149, 0.41234028339385986, 0.2928536534309387, 0.5025850534439087, -0.8624123930931091, 0.3387116491794586, 0.8024471998214722, -0.006789534818381071, -1.2184942960739136, -0.08043622970581055, 0.10164579004049301, -1.6671603918075562, -1.023768424987793, -0.040544673800468445, 0.2402266263961792, -0.4715738296508789, 0.6010242104530334, 2.4399585723876953, -0.6726784706115723], [0.9366111755371094, -0.8854852914810181, 0.37505003809928894, -0.438999742269516, 0.0077575272880494595, -1.4721349477767944, -0.8559423089027405, -1.3497329950332642, 1.564268708229065, -0.7754268646240234, -0.9823466539382935, -0.98697829246521, -0.01007602084428072, -2.049370288848877, 0.31953907012939453, 1.2285243272781372, -0.2108781486749649, 1.029225468635559, -0.789456844329834, -0.9913585782051086, -0.0511840283870697, -0.9227233529090881, -0.2099507749080658, -0.588081419467926, -1.7068909406661987, 0.3160148561000824, 0.8434039354324341, 0.48894232511520386, -0.8953970074653625, -0.35789090394973755, 1.3077930212020874, 0.22507143020629883, 1.024843454360962, -1.3387832641601562, -0.9769461154937744, -0.11482414603233337, -0.7512054443359375, -0.5003508925437927, 1.3634620904922485, 0.39531344175338745, 0.5436224341392517, -0.7922078371047974, 0.35684454441070557, -0.8005872964859009, 0.38613709807395935, -0.24152980744838715, 1.5264047384262085, -0.7714333534240723, 1.265034556388855, -0.9231410026550293, 0.7260614037513733, 1.0163769721984863, -1.018417239189148, -0.39173200726509094, -1.0550905466079712, -0.7936440706253052, 0.8789880275726318, -1.7019786834716797, 0.39640188217163086, -0.14224623143672943, -0.10202484577894211, -1.314733862876892, -1.1433968544006348, 0.09080404788255692], [0.1338326781988144, 0.9238380193710327, -0.3415398895740509, -1.3204498291015625, -1.4177252054214478, 1.5504934787750244, -0.6844936013221741, -0.2949449121952057, -0.728251039981842, -0.7572503089904785, -0.35240766406059265, -0.7448781132698059, 0.813249409198761, -0.537749171257019, 0.30794772505760193, 1.231830358505249, -0.08408484607934952, -0.9199607968330383, 0.5424095392227173, 1.2770158052444458, 0.7416590452194214, 0.22414807975292206, -0.10020937770605087, -0.9813003540039062, 0.2959020733833313, -0.8048553466796875, 0.29788830876350403, -0.9281551241874695, -0.428520530462265, -1.6582897901535034, -0.3723180890083313, 0.6437971591949463, 0.4163067042827606, 0.17975151538848877, -0.20104892551898956, -2.2906136512756348, 0.8080243468284607, -0.7242724895477295, -0.24701231718063354, -1.147192120552063, -0.9380442500114441, 0.40584567189216614, 1.1627992391586304, -1.8473670482635498, -0.22765502333641052, -0.12019232660531998, 0.20547394454479218, -0.7446975708007812, -0.9479475021362305, -0.2465268075466156, -0.3456084728240967, -0.6778568625450134, 1.6624525785446167, -0.3122672140598297, 1.3196336030960083, 1.7521157264709473, 1.50187087059021, 0.09648746997117996, 0.20121537148952484, -1.194683313369751, 0.7453426122665405, 1.3582782745361328, 0.293051153421402, 0.9552205801010132], [1.687294602394104, -0.3695283830165863, -0.3840978741645813, 0.30254530906677246, -0.528598964214325, -0.3529055714607239, -0.19366814196109772, 0.059829678386449814, 0.16710923612117767, -0.28060147166252136, -0.5648221373558044, -1.4184149503707886, 0.6251199841499329, 0.26293155550956726, 0.6068632006645203, 0.5428622961044312, 1.3266117572784424, 0.2743411958217621, 0.8394312858581543, 0.34566089510917664, -0.5393111705780029, -1.323275089263916, -1.7995924949645996, -0.5726429224014282, 2.404839277267456, 0.962843120098114, -1.609732985496521, -1.537200927734375, -0.1917610615491867, 0.782696545124054, -2.222874879837036, 0.7792182564735413, 0.6548777222633362, 1.986032247543335, 0.956299364566803, -0.19245021045207977, 1.4102445840835571, 1.0122106075286865, 0.367121160030365, 0.43795666098594666, -0.09241079539060593, 0.5372986197471619, 0.48792052268981934, 0.9167605638504028, -0.5669481754302979, 0.11671711504459381, -1.4248849153518677, -1.8781673908233643, 0.47998756170272827, 1.3916125297546387, 1.6740821599960327, 0.7100895643234253, -1.5594801902770996, 0.3443912863731384, -1.0102081298828125, 0.5884243845939636, -0.41716253757476807, -2.718726873397827, 0.08383554965257645, -0.6633791923522949, 0.7074916362762451, -0.5920949578285217, -0.5255776643753052, -0.9516556262969971], [-0.5525571703910828, 1.885953664779663, 0.2625335454940796, 0.5601891279220581, 1.0639585256576538, 1.0789189338684082, 0.5556715726852417, 0.15811392664909363, 0.5881219506263733, -2.3907113075256348, -0.2697718143463135, -1.3046563863754272, -2.0932717323303223, -0.09260568022727966, 1.1333091259002686, -1.610063076019287, -1.5073720216751099, -1.3581606149673462, 0.17267544567584991, -0.6439710855484009, -1.0071475505828857, -1.7142525911331177, -0.6128356456756592, 1.659334659576416, -1.515080213546753, -0.4340152144432068, 0.29052457213401794, -0.4030072093009949, 2.1516354084014893, -1.524489402770996, 0.070683054625988, -0.6850983500480652, 0.5345600247383118, -0.827265739440918, -0.3665471076965332, -0.3079988658428192, -0.39810889959335327, 0.15402469038963318, -1.1032813787460327, -0.9964747428894043, -0.11193788051605225, -1.5009547472000122, -0.4775351881980896, -0.4975467026233673, 0.7802843451499939, -0.9999415874481201, -0.5375256538391113, -1.2480394840240479, -1.1998722553253174, 0.5991045832633972, 1.2019590139389038, -0.904805600643158, 0.2893716096878052, -0.5443769693374634, -0.12304159253835678, 0.18416793644428253, -1.1759978532791138, -1.4615486860275269, 1.2478375434875488, 1.7055718898773193, -0.37406182289123535, 0.4667517840862274, 0.5929561853408813, 0.6532722115516663], [0.06294681876897812, -0.23708327114582062, -0.36209461092948914, 1.644592523574829, 0.37370508909225464, 0.04153798520565033, 2.1339597702026367, 0.5044956207275391, 1.8344088792800903, -1.634729027748108, 2.0033833980560303, -0.40907955169677734, 0.5505118370056152, -0.6362380385398865, -0.23214085400104523, -0.6297978758811951, -1.3705984354019165, -1.5553927421569824, 1.231868028640747, -1.1425739526748657, -0.1546907275915146, 0.30935320258140564, 0.6211996078491211, -1.0343159437179565, -0.7974932193756104, -0.04919194057583809, 2.0449695587158203, -0.47101089358329773, 1.7273812294006348, 1.278581142425537, -1.0021005868911743, -2.683337926864624, -0.0978824719786644, 0.338895708322525, 0.6146821975708008, 0.42087802290916443, 0.8280410170555115, -1.214831829071045, 0.505497395992279, -0.27814340591430664, -0.46761617064476013, -0.2663474977016449, -0.2776378095149994, -0.3277176022529602, -0.2708238661289215, 1.8371222019195557, -1.7534884214401245, -0.4953508973121643, 0.04070663824677467, -1.117939829826355, -0.31699684262275696, -2.352201461791992, -0.6076903343200684, 0.20377643406391144, -1.0263866186141968, -0.30479320883750916, -0.5560803413391113, -1.177445650100708, -0.11149398982524872, -0.536985456943512, -0.8093336224555969, -0.281227707862854, -0.2041371464729309, 1.7407739162445068], [1.0384621620178223, 0.7769439816474915, 0.583882212638855, 0.8604732751846313, -0.5123459696769714, 1.2158386707305908, 1.0069222450256348, -1.528580904006958, -1.9221224784851074, 0.5123364925384521, 0.2060731202363968, -0.02221163921058178, -0.1812947541475296, 0.5575149655342102, 0.42548301815986633, 1.3703926801681519, 0.27819836139678955, 0.47038155794143677, -1.6918028593063354, 0.41824910044670105, 0.8753966689109802, -1.189056396484375, 2.3558573722839355, -0.35106945037841797, -1.170506238937378, 0.7069692611694336, -0.5051031112670898, -1.2760236263275146, -0.9322372674942017, 0.405310720205307, -0.6911371350288391, -0.40043359994888306, 1.4022574424743652, -0.30957531929016113, -1.1969795227050781, 1.2383179664611816, -0.267300546169281, -1.1844130754470825, 0.6824241876602173, -0.050161320716142654, -0.19252611696720123, -1.5679950714111328, -0.9223126769065857, -1.641058087348938, -0.9103367328643799, 1.0676310062408447, -1.6166671514511108, -0.4064510464668274, 0.5792626142501831, -0.15972422063350677, 1.2199567556381226, 0.03809477016329765, -0.8289009928703308, -0.2169117033481598, 1.2896454334259033, -0.2974202334880829, 1.309098482131958, 0.4393274188041687, 0.4931964874267578, 0.45360106229782104, -2.313312292098999, 0.6949023604393005, -0.602832555770874, 2.2969002723693848], [-0.32747384905815125, -0.07567834854125977, -0.05020746961236, 1.4405783414840698, -0.28612765669822693, -0.6700118184089661, -1.590425729751587, -0.18907895684242249, -1.641676902770996, -2.1366705894470215, 1.1167123317718506, -0.8595507144927979, -1.6038326025009155, -0.7895679473876953, -1.6969940662384033, -0.0963786318898201, -1.0126352310180664, -1.288495421409607, -0.7476866245269775, -0.44288694858551025, -1.1166104078292847, -0.5434961318969727, 1.6207656860351562, -0.8969213962554932, -0.2558625340461731, 1.9554046392440796, 0.21268720924854279, -0.621330201625824, -0.848969042301178, 1.6034907102584839, -1.0242024660110474, -0.11909432709217072, 0.3736153542995453, 0.1999834179878235, 1.3368582725524902, 0.1783401370048523, -0.8517546653747559, 0.9316439628601074, -1.8727442026138306, 1.7205119132995605, 0.548895537853241, 0.13676339387893677, 0.13049575686454773, -1.389243721961975, 0.09431719034910202, -0.00780827272683382, 1.6576405763626099, 0.03892489895224571, 0.08558235317468643, 1.2643641233444214, 1.007475733757019, -0.24670512974262238, 1.1009001731872559, -0.6180546283721924, -0.5465918779373169, 0.11828087270259857, 0.07511505484580994, -0.7021095752716064, -0.940126359462738, 0.7743459939956665, -0.588253915309906, 0.6094754934310913, -0.9909142255783081, 0.7192036509513855], [-0.33688220381736755, 0.7695549130439758, -0.29987281560897827, 0.8573645353317261, -0.9840744733810425, 0.1188674122095108, 0.7072440385818481, 1.5246658325195312, -0.6594833731651306, 0.5193689465522766, -1.4430283308029175, -1.4615857601165771, -2.7005906105041504, -1.0501166582107544, 0.34054461121559143, 2.069143772125244, -0.13921618461608887, 0.06641262024641037, -0.4169015884399414, 1.6285550594329834, -1.9377901554107666, -0.05537843704223633, 0.28855884075164795, -0.9662097692489624, -1.4198884963989258, -0.46480515599250793, -1.5175201892852783, -0.759515106678009, 0.8492456674575806, 2.773629903793335, -0.009455900639295578, -0.8000336289405823, -0.02681192196905613, 0.30819982290267944, 0.4240248203277588, -1.2596644163131714, 0.13066738843917847, -1.311048150062561, 1.5831927061080933, 0.3184531629085541, -0.9122394919395447, 0.41602054238319397, -1.5975072383880615, -0.8920093774795532, 1.2526872158050537, -1.3284741640090942, -0.797459065914154, 1.8542052507400513, -1.155792474746704, 0.2372356504201889, -2.2869679927825928, -0.10108505189418793, -0.48677319288253784, 0.41213950514793396, -0.07152999937534332, 1.2328588962554932, 1.0115689039230347, -0.7262781262397766, -0.35640937089920044, -0.8299188017845154, 0.4776449203491211, -0.570611298084259, 1.225372314453125, -0.812236487865448], [-1.159295678138733, -0.4809406101703644, 0.5400351881980896, 0.6263138055801392, -0.7688073515892029, 0.08236744999885559, -0.32088232040405273, -1.7438205480575562, -2.3304827213287354, -1.5914217233657837, 0.06605193763971329, 0.4293416440486908, 1.120153784751892, 0.9240690469741821, 0.20036442577838898, -1.9852170944213867, 0.5857858061790466, -1.6426104307174683, 1.5674498081207275, -0.6047929525375366, 0.45566344261169434, -0.32693594694137573, -0.4982087016105652, -0.9915704727172852, 0.9169235229492188, 0.3275901973247528, 1.9507579803466797, 0.9945557713508606, -0.006655044388025999, 0.3263015151023865, 0.2698933780193329, -0.9905296564102173, -1.7110651731491089, 0.1884920746088028, -0.06437364965677261, 0.3249814510345459, 0.19019579887390137, 0.7042751908302307, -0.8875200748443604, -1.7441999912261963, -0.6010904908180237, -1.0496137142181396, 0.49151474237442017, -1.1698909997940063, 0.21957144141197205, -0.0649343729019165, -0.6920017004013062, -0.3562760651111603, 1.0967175960540771, -0.16331811249256134, 1.5246357917785645, -0.12708355486392975, 1.2978893518447876, -0.19869625568389893, -1.9932414293289185, -0.7416605949401855, 0.7771335244178772, 0.40668734908103943, 2.1471853256225586, 0.7564374804496765, -1.9885090589523315, 1.5051124095916748, 0.8484559059143066, -0.019138900563120842], [-1.9848946332931519, 1.2074693441390991, 0.7417691946029663, 0.3472832441329956, 0.22701255977153778, 0.2592228055000305, 0.9806197285652161, 0.4626207947731018, -0.7295914888381958, -0.18167394399642944, 0.3941477835178375, 0.08563325554132462, -0.7955163717269897, -1.4337621927261353, -1.155099868774414, 1.0483673810958862, 0.23485873639583588, 0.7470228672027588, -0.6735570430755615, -1.0155425071716309, -0.4358336925506592, -0.6383450031280518, -1.1374459266662598, -0.04970547556877136, -0.7812938690185547, 1.2730728387832642, 1.5859259366989136, 0.16257819533348083, -1.6194062232971191, 2.7312893867492676, -0.6293525695800781, 0.04986022040247917, -0.24802660942077637, 0.9768851399421692, 0.19922864437103271, -0.8770327568054199, -1.5483882427215576, 2.7572739124298096, 1.0435649156570435, 0.4139961302280426, 0.08071301132440567, 1.3285166025161743, 1.6271774768829346, 0.4445774555206299, 0.34180542826652527, -0.8399763107299805, 1.483580470085144, -0.4185483455657959, 1.0559651851654053, 0.7075111269950867, 2.4421792030334473, -0.5712615251541138, 0.07408538460731506, 0.8518620133399963, 0.27115771174430847, 1.305485725402832, 2.2818572521209717, 0.16827310621738434, 0.2317306399345398, 2.7188475131988525, -0.235466867685318, -0.18195375800132751, -0.7459654808044434, -1.417189598083496], [-0.5633021593093872, -0.9669696092605591, -1.912758708000183, 0.5703073143959045, -0.3360179364681244, -0.17059484124183655, 0.14362697303295135, 0.16036823391914368, 1.6415488719940186, -0.5950635671615601, -0.6621244549751282, -0.4929002821445465, 1.6682054996490479, 1.1948494911193848, 0.4948952794075012, 1.724366307258606, 0.5695120096206665, -0.4387882649898529, 1.0087577104568481, 0.06689552962779999, 0.26696038246154785, -1.149073600769043, -0.9585459232330322, -0.25763678550720215, -0.7918482422828674, 0.943440854549408, 0.47601914405822754, -0.7356566786766052, 1.5207313299179077, -0.9615020751953125, 2.623296022415161, 1.796891689300537, -0.5300630331039429, 1.7504267692565918, -0.2702270746231079, -0.3585006594657898, 0.6442528367042542, 0.9790849685668945, 0.27811935544013977, -0.10794272273778915, 1.001922845840454, 1.4152384996414185, -1.023300290107727, 1.432732343673706, 0.548783004283905, 1.2747347354888916, -0.13124138116836548, 0.47610723972320557, 0.8877947926521301, -0.5682471990585327, 0.3481481969356537, -0.1246945708990097, -0.17846255004405975, 0.5286492109298706, -0.518984317779541, 0.37276554107666016, 0.4843268096446991, 1.4126009941101074, -0.6841533184051514, 1.650858998298645, 1.3027504682540894, -1.2905068397521973, -0.24567219614982605, -0.6009789705276489], [1.647722601890564, 0.7069717645645142, -0.3031754195690155, 0.6747696399688721, 0.7083734273910522, -0.9732508659362793, 0.40663912892341614, -1.0970808267593384, 0.41718825697898865, -0.1840231567621231, -1.0769824981689453, 0.7365646958351135, -0.15171806514263153, 0.6452077031135559, -0.6963618993759155, -0.1459057480096817, -2.0007858276367188, 0.9410354495048523, 0.27664411067962646, 0.8727189302444458, -0.43166014552116394, 0.13253773748874664, -0.03459019586443901, 0.7750501036643982, 0.17368820309638977, 0.7045741081237793, 0.8443848490715027, -0.06620465219020844, 0.8213092088699341, -0.17004743218421936, -0.37998631596565247, -0.3892723023891449, -0.45611169934272766, -0.8114275932312012, -0.3657410442829132, -0.5399393439292908, -1.0556572675704956, -1.4999111890792847, -1.904006838798523, 0.16316072642803192, 0.7552491426467896, 0.12657004594802856, 0.9231284260749817, 1.3094033002853394, 0.5748882293701172, -2.3788061141967773, -0.7283603549003601, 0.20814140141010284, 1.8386240005493164, 0.5566139817237854, -0.8930507898330688, -1.1807466745376587, -1.372594952583313, -0.09735145419836044, 0.39244359731674194, -0.7419530153274536, 1.82920503616333, -0.2354755699634552, -1.1472806930541992, 0.5200322866439819, 0.4371107816696167, 0.34144651889801025, 0.32232865691185, -0.27200040221214294], [-1.3972773551940918, -2.2961905002593994, -1.5196740627288818, 2.7702198028564453, 0.8045220971107483, 1.9782179594039917, -0.010109751485288143, -0.4543960392475128, 0.1994980424642563, 0.6249259114265442, -0.8135095834732056, -0.01600576378405094, -1.3644315004348755, 1.956712245941162, -0.8049582242965698, -0.5139743685722351, -0.5313749313354492, -0.23120732605457306, 0.4623070955276489, -1.7963770627975464, -0.695606529712677, -0.42499226331710815, 0.4063669443130493, -0.9861975312232971, -1.4768776893615723, -1.0527857542037964, 0.5937036275863647, 0.29716411232948303, 0.9756380915641785, -0.3738570213317871, 0.8270858526229858, -0.09006792306900024, 0.45325836539268494, 0.6556236147880554, 1.558222770690918, -0.2512066066265106, 0.5378245115280151, -0.5453037619590759, 1.5444945096969604, 0.4061441123485565, -1.6500210762023926, -0.3255673348903656, 0.11909948289394379, 0.9323405623435974, 0.3784388303756714, 0.49003028869628906, -1.4017748832702637, -0.008627476170659065, 0.31711408495903015, -0.8766129612922668, 0.12207486480474472, -0.16281116008758545, 0.42342978715896606, -1.613585114479065, 0.5263318419456482, -2.1398262977600098, -0.31204086542129517, 0.3856496512889862, 0.6751198768615723, -0.32313069701194763, 1.5491975545883179, 1.8753117322921753, 0.652193009853363, 2.9316728115081787], [-1.0499229431152344, -0.19114409387111664, 0.2784581780433655, -0.9125614762306213, 0.5130013227462769, -0.15100868046283722, -0.038548726588487625, -0.5969119071960449, 0.07213874161243439, -0.018361834809184074, -1.50743567943573, -0.2365429401397705, 1.2662376165390015, 0.6762198805809021, -0.08318775147199631, 0.6050717830657959, 0.6126081347465515, -0.19963233172893524, 1.5805617570877075, -0.7568455338478088, -0.8817309737205505, 0.15647141635417938, -0.0043501718901097775, 0.05712669715285301, -1.3362504243850708, 0.7342341542243958, 0.7487972974777222, -1.160248041152954, -2.206561803817749, -0.9755314588546753, 0.9778390526771545, -1.2517359256744385, 0.5640154480934143, -0.8341869711875916, -0.2665739953517914, -1.3743194341659546, 0.5365082025527954, 1.341709852218628, 0.4589403569698334, 0.4826613962650299, -0.52105712890625, -0.13063617050647736, -0.6673818826675415, 1.0126644372940063, -0.9552788138389587, 0.9799983501434326, 0.001967539545148611, 0.544083833694458, 0.6338742971420288, -1.5717757940292358, -0.21921998262405396, -0.26778939366340637, 2.2836263179779053, 1.0402486324310303, -0.8356233239173889, -0.02279955893754959, -0.2563754916191101, 0.36751505732536316, -0.48902085423469543, -0.6557946801185608, 0.7903628349304199, -0.8452133536338806, 0.1305651068687439, -0.3826758563518524], [-0.30377909541130066, 0.6549332141876221, -1.1729599237442017, -1.994444727897644, 1.2319239377975464, -2.1290271282196045, -0.7348018884658813, 1.1697595119476318, -1.1395975351333618, 1.3873180150985718, 0.38040852546691895, -0.5367206335067749, 0.853471577167511, 1.401902675628662, -0.4299665093421936, -2.2080516815185547, -1.2840596437454224, 0.9089029431343079, -2.8243086338043213, 0.45585766434669495, 1.655299425125122, 1.2060251235961914, 0.8917654752731323, -0.12245449423789978, 2.1657791137695312, -2.1689953804016113, 2.402949810028076, 0.953185498714447, 0.25963515043258667, 1.701374888420105, -1.0528305768966675, 0.3900071978569031, -0.0403103232383728, 1.0565013885498047, 1.0277718305587769, -0.9308738112449646, 0.6781372427940369, 0.8700153231620789, -1.2921875715255737, -0.4096960425376892, 0.32572925090789795, 0.19578109681606293, -0.8768722414970398, 1.1871209144592285, -2.231121778488159, 1.8084304332733154, -0.05960385128855705, -1.166948914527893, -1.220359444618225, -0.9402364492416382, -1.8321092128753662, 0.011448081582784653, -0.43408575654029846, -2.0198349952697754, -1.7997913360595703, -0.06666584312915802, 0.8821969628334045, -1.3075231313705444, -1.1655025482177734, 1.2474040985107422, 0.34788942337036133, -0.256503701210022, 1.4164875745773315, -1.1944626569747925], [1.3125840425491333, 0.6924299597740173, 0.3895246684551239, -0.17896609008312225, 1.2216264009475708, -0.0294882133603096, 0.48838403820991516, 0.2884990870952606, 1.274330735206604, -0.8100646734237671, 1.550884485244751, -0.4839403033256531, -0.627964973449707, 0.1548997312784195, 0.1963973045349121, -0.5229031443595886, -0.5140219926834106, -0.8636090755462646, 0.9222449660301208, 0.13739743828773499, -2.344381332397461, 0.02575182542204857, 1.0259288549423218, 0.37796685099601746, 1.7472171783447266, -0.5144276022911072, -1.7243841886520386, -0.5570849776268005, 0.0055588623508811, -0.013777966611087322, 0.1528589278459549, -0.35179632902145386, 1.010223388671875, 0.7151739597320557, -1.174328327178955, -0.5010323524475098, -1.0687763690948486, -0.04147215560078621, 1.1501789093017578, -0.9832106828689575, 1.4059661626815796, 0.20805515348911285, 1.470046877861023, -1.0548804998397827, -0.4914619028568268, -1.4509906768798828, 0.328357070684433, -0.04918864369392395, 1.1114249229431152, 0.1819409281015396, 0.6371227502822876, 0.037153687328100204, -0.665598452091217, 0.8213473558425903, -1.7008001804351807, -0.3054853081703186, 0.12346439063549042, -1.3983696699142456, -0.39034029841423035, -1.079070806503296, -0.4927889108657837, -1.01157546043396, 0.06300247460603714, 0.4333028495311737], [-0.2848449647426605, -0.17788858711719513, 0.19772562384605408, -0.5662622451782227, -0.19610188901424408, 0.03280195593833923, -0.08988092839717865, 0.3246215879917145, 0.1908537596464157, -0.10569042712450027, 0.41530951857566833, -0.14256682991981506, 0.5156824588775635, -1.6294775009155273, -1.377942442893982, -0.22563159465789795, 0.5690838694572449, -1.445385456085205, 0.42215466499328613, 1.2408664226531982, 0.0358312726020813, -1.0932525396347046, -0.37661564350128174, -0.944155216217041, 1.5078281164169312, -0.5506126880645752, -0.8699629902839661, -0.06308028846979141, -0.006901678629219532, -0.6829233169555664, -0.18887872993946075, -0.763667106628418, -0.33651798963546753, 0.6505120396614075, 1.722965955734253, -0.13080719113349915, -0.437825471162796, 0.44321465492248535, 0.28069284558296204, 0.9131321907043457, 2.0960752964019775, 0.22927729785442352, -0.5490432977676392, 0.15233765542507172, -1.2580560445785522, 0.462722510099411, -0.3965007960796356, -1.4769724607467651, -1.6234296560287476, 0.26237478852272034, -1.1973403692245483, -0.9794782400131226, 2.093461513519287, -0.9254292845726013, 0.7442482113838196, -1.0086987018585205, -1.5446600914001465, -0.3075192868709564, 0.7371686697006226, -0.4142457842826843, 0.05086151883006096, 1.8781137466430664, 0.7693321108818054, 2.0189309120178223], [-1.0638717412948608, -0.09441439807415009, 0.536934494972229, -1.029956579208374, 0.8134281635284424, 0.23011983931064606, 1.164609670639038, -0.12124542891979218, -0.37254419922828674, -2.5056803226470947, -0.5257464647293091, -1.7266336679458618, 0.6431931257247925, -1.104000210762024, 0.533664345741272, -0.5744295716285706, -0.4857821464538574, 1.3425828218460083, -1.7221156358718872, 0.9602057933807373, 2.3297557830810547, -1.3468509912490845, -0.4001849591732025, 0.4147963523864746, 0.24027839303016663, 1.6581506729125977, 1.0503562688827515, -0.7116008400917053, 1.6150935888290405, 1.2487472295761108, -0.0452394112944603, -1.8044995069503784, 0.7483428120613098, 0.0857076570391655, -1.6730554103851318, 0.17861294746398926, 0.2814517021179199, 1.0059155225753784, 0.03393905982375145, -0.2811923921108246, 1.5626941919326782, 0.28670212626457214, 0.6740130186080933, 1.2959150075912476, 0.48666268587112427, -0.3540928363800049, -1.2092639207839966, 1.7367717027664185, -0.5195618867874146, -0.5161764621734619, 0.8691173791885376, 1.4164447784423828, -0.6132881045341492, 2.1379754543304443, 0.22064341604709625, -0.4732144773006439, 0.6821448802947998, -0.1688428819179535, 1.2558950185775757, -0.6140464544296265, -0.23270036280155182, 0.47125571966171265, -1.6635723114013672, 0.6287883520126343], [0.2171199470758438, 1.7195320129394531, -0.5430401563644409, 0.92384272813797, -0.23051530122756958, -0.8128913044929504, -0.04921744018793106, 0.972577691078186, -1.0932109355926514, 0.21568888425827026, 1.0266145467758179, -0.3213498592376709, 0.26247134804725647, 1.4782471656799316, 0.4707653522491455, -2.5182571411132812, 0.4318903982639313, -0.6902027130126953, 0.9872121810913086, -0.4124988913536072, -1.6111116409301758, -0.72845059633255, -0.8214185833930969, 0.19120539724826813, 0.4894695580005646, -0.2516074776649475, 0.7032153010368347, -0.8854852914810181, 0.1349230259656906, -1.262270450592041, -0.8576423525810242, 0.9748740196228027, 2.0046815872192383, 1.0620479583740234, -0.9538565874099731, 1.0022647380828857, -1.0187523365020752, -1.8851813077926636, 0.7701168060302734, -0.42838960886001587, -0.7861524820327759, -0.4464294910430908, 0.10297452658414841, -2.5334503650665283, -0.13447508215904236, -1.3804091215133667, -2.535184383392334, 0.015676157549023628, -0.029007717967033386, 0.5187922120094299, 0.5379298329353333, -0.6637306809425354, 1.2333755493164062, -2.067499876022339, 1.2124271392822266, 0.25805217027664185, -0.7978287935256958, -0.8718922734260559, 0.7954096794128418, 0.4900807738304138, -1.7724529504776, 0.5078617930412292, 1.2842402458190918, 0.10828738659620285], [0.22838200628757477, -0.827711820602417, 0.3908345699310303, 1.9766157865524292, -2.059617757797241, 0.9982562065124512, -1.3144537210464478, -0.7381969690322876, 0.24105148017406464, -0.4759899973869324, -1.4760547876358032, 1.2719804048538208, 0.6819625496864319, -0.8708105087280273, 1.0272057056427002, 0.14135004580020905, -0.9943691492080688, 0.10039108246564865, 1.233513593673706, 1.0403225421905518, -0.9935606122016907, -0.1782875955104828, 0.8548983931541443, -0.23191770911216736, -0.15800665318965912, -0.10889185965061188, 0.9587097764015198, -0.704408586025238, 1.1194841861724854, -1.9708633422851562, 0.2728893756866455, 1.3905032873153687, -0.20337121188640594, 1.9999759197235107, -1.670404076576233, -1.493184208869934, -0.12578044831752777, 2.1295387744903564, 0.20838655531406403, -0.9193466901779175, -0.45001253485679626, -0.6970165967941284, -1.7729580402374268, 0.341248482465744, -0.2640707194805145, -0.6719123721122742, -1.3910155296325684, -0.9766794443130493, -2.0137388706207275, -0.35287004709243774, -1.8257397413253784, -0.2166070193052292, -0.3136606514453888, 1.5076568126678467, -1.3107572793960571, 0.47061699628829956, -0.22254827618598938, 0.7744345664978027, 1.4733524322509766, 0.6361398100852966, 0.30660200119018555, -0.4758566915988922, 0.21188655495643616, 0.5707165598869324], [-0.9195995926856995, -1.3159574270248413, -0.9751169085502625, 0.8099618554115295, 0.24283505976200104, -0.5816928148269653, 0.8881368041038513, -0.9748424887657166, -0.5274879336357117, 0.12894511222839355, 0.3160547912120819, 1.247228741645813, -0.07065800577402115, 0.2140308916568756, 0.07307969033718109, 1.1930205821990967, -1.2015790939331055, 0.2697097957134247, 0.22394154965877533, 0.8346126079559326, 0.04234210401773453, 1.5834839344024658, -1.0544856786727905, 0.7492879033088684, -1.0158472061157227, -1.0555299520492554, 0.18462809920310974, -2.20294451713562, 0.8975893259048462, 0.16034150123596191, -0.5776322484016418, 1.6848541498184204, -1.3408100605010986, -0.42067474126815796, -0.1865973174571991, 1.5052275657653809, 1.7363070249557495, 0.971727192401886, -0.41272249817848206, -0.030505763366818428, 0.1875871866941452, -0.28732940554618835, 0.1108141615986824, 0.957545816898346, 0.3870304822921753, 0.7014190554618835, -0.5128506422042847, 0.18750546872615814, -0.3837587535381317, -0.20117878913879395, 0.34034982323646545, 0.3389452397823334, -0.5625767707824707, -0.5058916807174683, -0.8547398447990417, 1.1452622413635254, 1.2566124200820923, -0.3039827048778534, -0.5870015621185303, -0.9108587503433228, -0.48829877376556396, 0.8696153163909912, -1.1075092554092407, 1.3412913084030151], [0.5940960645675659, 0.01650499552488327, 1.8325153589248657, -0.6602610945701599, -0.2918894290924072, -0.08155591040849686, -0.48596838116645813, -0.18089087307453156, -0.8690942525863647, -0.5974410772323608, -0.36108633875846863, 0.6689238548278809, -1.8555002212524414, -1.307708740234375, -1.1675721406936646, 0.15220700204372406, 0.06417688727378845, 0.707798182964325, -1.208234190940857, 0.3804168105125427, -0.6482396721839905, 0.6079314947128296, 2.3638525009155273, -0.7505815625190735, -0.329323410987854, 1.5015997886657715, 1.1317946910858154, 0.6271942257881165, -0.3000617027282715, 0.6157172322273254, 1.0273518562316895, 0.847276508808136, 0.9298922419548035, -2.0942771434783936, 0.8635226488113403, 0.679027795791626, -0.2691119611263275, -0.8876274228096008, 1.404936671257019, -0.7475689649581909, -0.9553964138031006, 1.6441261768341064, -0.4918363094329834, -0.36368635296821594, -1.5418775081634521, 0.7251649498939514, 0.04653545841574669, -3.139441728591919, -0.7460017204284668, 0.5187495946884155, 0.21822650730609894, 1.2174441814422607, 0.7071345448493958, -0.16245780885219574, -0.8059359192848206, -1.79282546043396, 0.11463091522455215, -0.029568180441856384, -0.33823877573013306, -0.601271390914917, -0.0901184156537056, 0.13015519082546234, -1.7420254945755005, 0.13341452181339264], [2.231717348098755, 1.336736798286438, -0.39142704010009766, 1.1976336240768433, 0.10069753229618073, 2.8653616905212402, -0.998267650604248, 0.9314530491828918, 0.3272949755191803, -0.3113114833831787, -0.38444623351097107, -0.8836833238601685, -0.6343280673027039, 2.6012001037597656, 1.9573187828063965, 0.08163140714168549, -0.6865573525428772, 0.8261312246322632, 0.23466037213802338, -0.6800384521484375, -0.09149864315986633, -1.3247774839401245, -0.6732028126716614, -0.1730738878250122, 0.8011190295219421, -0.7278223633766174, 1.2382407188415527, -1.317640781402588, 1.2079472541809082, 0.3184301257133484, -0.24068252742290497, -0.7711659669876099, 0.7411898374557495, -3.0244905948638916, -0.5769672393798828, -0.12766064703464508, -0.44798195362091064, 1.5414479970932007, 0.015627074986696243, 0.3175268769264221, 2.4084715843200684, -0.3005521595478058, 0.3655541241168976, 0.4850144386291504, -1.0208590030670166, 1.274297833442688, 0.8093311190605164, -0.8809198141098022, -0.5464199185371399, -0.25516557693481445, 0.7633923292160034, -0.11782889813184738, -2.1072983741760254, 0.643633246421814, -0.23064428567886353, 0.4563755393028259, 0.03981785848736763, 0.4267975091934204, -0.6244882941246033, -0.5146481990814209, 0.1256607174873352, -0.5417109131813049, 0.003971126861870289, 1.0948944091796875], [-0.8124380111694336, -0.408010333776474, 0.6049730777740479, 1.561855673789978, 0.19198621809482574, 0.2180071473121643, 0.8996296525001526, 1.485641360282898, -0.5288197994232178, 1.0965675115585327, -1.0372660160064697, -1.4273689985275269, 1.187347412109375, -2.408097505569458, 0.359785795211792, 1.7965993881225586, -0.8355715870857239, -0.016491113230586052, -1.9594907760620117, 0.47979578375816345, -0.32428696751594543, -1.6084226369857788, -1.2932194471359253, -0.414905309677124, -1.5870158672332764, 0.9360615611076355, 0.9658619165420532, 1.0358026027679443, 0.23710742592811584, 0.01065085269510746, 0.713860273361206, 0.19855447113513947, 1.2310688495635986, 0.39304137229919434, -1.226905107498169, -1.7092456817626953, 0.07766468077898026, -1.7713642120361328, 1.2837121486663818, -1.1912115812301636, 1.2483546733856201, 0.6826842427253723, 1.3131211996078491, 0.2399301528930664, -0.2573689818382263, -1.257523536682129, -0.24756473302841187, -1.0511010885238647, 0.11312053352594376, -0.8032824397087097, -0.5507412552833557, 0.1085839569568634, -1.6702581644058228, 1.1149572134017944, 0.35240739583969116, 0.35112300515174866, 0.37213706970214844, 0.1567915678024292, -0.49215734004974365, 1.4592634439468384, -0.4779086112976074, -1.0329960584640503, -1.147089958190918, 0.023827875033020973], [0.20422610640525818, 0.8913582563400269, -0.06268726289272308, 1.3529495000839233, 0.7264833450317383, 0.31049826741218567, 1.0019644498825073, 1.5194472074508667, -0.5352991819381714, -1.3039419651031494, -0.001477471669204533, 0.2946312725543976, 1.2135868072509766, -1.8046079874038696, -0.09139521420001984, 0.05228283256292343, 0.9901725053787231, 0.8286267518997192, 0.5327038764953613, 1.9400380849838257, -0.6319507360458374, 0.4057467579841614, 1.2419692277908325, 0.2982082664966583, -0.3860962986946106, -1.793487310409546, -1.5986249446868896, -0.4545093774795532, -0.7299588322639465, -0.18532951176166534, -1.608189344406128, -1.738739013671875, -0.8022727370262146, -0.840485692024231, 0.3905768394470215, 0.21893872320652008, -0.18935512006282806, 0.1896442174911499, 0.24133841693401337, 0.02995738759636879, -1.1566377878189087, -1.2574008703231812, -0.6156617403030396, 1.4985876083374023, -0.9103338718414307, -0.6698659658432007, -0.8841938972473145, 0.6665899753570557, -1.0892566442489624, -1.2801178693771362, 0.0973755419254303, -0.3769178092479706, 1.1163101196289062, -1.2028464078903198, -0.8644185662269592, 0.6836968660354614, 1.0228074789047241, -0.6693724989891052, -0.39165443181991577, -1.6336395740509033, 0.17343871295452118, 1.4483119249343872, 0.16933462023735046, -0.7019413709640503], [1.0931469202041626, 1.0927642583847046, 0.14056818187236786, 1.1782505512237549, -0.09228389710187912, -1.0310149192810059, 0.2462392896413803, -0.5044886469841003, -0.28014692664146423, 0.2299916297197342, -0.30129826068878174, -0.013828715309500694, 0.9285339713096619, 0.9363844394683838, 1.070947289466858, -0.8242563605308533, 1.4898144006729126, -0.0447995588183403, -1.0765316486358643, -0.30151617527008057, -0.9890342950820923, 0.009088444523513317, 0.625741183757782, 0.32903504371643066, -1.0436630249023438, 0.8186497688293457, 1.1901732683181763, -0.7242195010185242, 0.6910807490348816, 0.10039462149143219, 0.5633385181427002, -1.8506581783294678, 0.7770333290100098, 0.025105560198426247, -2.000544548034668, 0.290056437253952, -0.21896106004714966, 0.3810849189758301, -0.48219332098960876, 0.3364025354385376, -0.8039164543151855, 0.698799729347229, 0.8886844515800476, 1.5038889646530151, 1.6734092235565186, 1.8330276012420654, 0.1260145753622055, 0.1286129504442215, -1.1901625394821167, 1.8349744081497192, -1.2399773597717285, -0.4624328017234802, 0.22331462800502777, 1.5556873083114624, 1.5643974542617798, 0.4032750725746155, -0.0857936441898346, -0.3864705264568329, 0.2661431133747101, -1.167249083518982, -1.050935983657837, -0.6879386305809021, -0.000384566024877131, -0.3410930037498474], [1.4530706405639648, -0.22552897036075592, 0.0496525838971138, -1.5412970781326294, 2.1772475242614746, -0.28876569867134094, 1.2241309881210327, 2.4435670375823975, 1.1823538541793823, 0.16181153059005737, 0.2661694884300232, -0.7842700481414795, 0.5569822788238525, -1.6815465688705444, 0.6828132271766663, 0.8415512442588806, 0.21306025981903076, -0.3220854699611664, -0.12952734529972076, 0.3038218021392822, -0.9117254018783569, 0.5168407559394836, 1.1148619651794434, -1.9429547786712646, 0.35500046610832214, -1.005831241607666, -1.3130332231521606, 0.20100650191307068, 0.4972299635410309, -0.8219382166862488, -1.431357741355896, -0.7802165150642395, 0.7082043886184692, 1.0389364957809448, -0.7455862164497375, -1.0438766479492188, -0.4923326373100281, -1.1519745588302612, -0.679344654083252, -0.7641732096672058, -1.2918802499771118, 1.7992420196533203, -0.5663154125213623, -1.2878801822662354, -0.38690412044525146, -0.35832715034484863, -0.013011744245886803, -1.8555727005004883, 0.7102295756340027, -0.8391757011413574, 0.44008535146713257, -1.3181053400039673, 2.1919267177581787, 0.8348898887634277, 0.14323130249977112, 0.447890043258667, -0.8437216281890869, -1.07244074344635, 0.19991043210029602, 1.0969855785369873, 0.03452315926551819, -0.10036451369524002, 0.09729847311973572, -0.7411901950836182], [-0.8532389998435974, -0.9538493752479553, 1.1703993082046509, -0.16406399011611938, -0.43080219626426697, 1.0959237813949585, 0.22163614630699158, -0.00016881980991456658, 0.2075091004371643, 0.6310780644416809, 0.2106839269399643, 0.5667256712913513, 0.5036168098449707, -0.7581723928451538, 1.5937113761901855, 1.70004403591156, 0.03484005108475685, 0.8050689697265625, -1.124287724494934, -0.01039429847151041, -0.5932032465934753, -1.4503344297409058, -0.2739778161048889, -0.6091724634170532, -1.9023135900497437, -1.9626981019973755, -3.2400763034820557, 0.45469048619270325, 0.27336418628692627, 0.4637618660926819, 1.1951661109924316, -0.36496588587760925, 0.22178815305233002, -0.3433147966861725, 1.2369452714920044, -0.9895513653755188, -0.09789947420358658, 0.19143685698509216, -0.011205330491065979, 1.6949732303619385, 1.0237281322479248, -0.23110593855381012, -0.8061947226524353, -0.6798840761184692, -1.640057921409607, 0.23237881064414978, 0.8155614137649536, -0.6054879426956177, -0.7182947993278503, 0.8396226763725281, 0.24426493048667908, -0.8817841410636902, 0.6881186366081238, -0.9160533547401428, -1.3301095962524414, 0.7761979699134827, -0.01335077453404665, 1.5051765441894531, -0.3898315131664276, -0.09049293398857117, -0.3717014789581299, -1.0534080266952515, -1.2104052305221558, 0.0563473105430603], [1.732543706893921, -1.0099960565567017, 0.8643197417259216, 2.6065962314605713, 2.1620543003082275, -0.5496779680252075, -0.2040136158466339, 2.3787736892700195, -1.5316423177719116, 0.7001320123672485, 1.6822924613952637, 0.053635455667972565, -0.6025007367134094, 0.6752581596374512, -0.007829072885215282, 0.020656993612647057, -0.44795656204223633, 0.3182470202445984, -1.2486896514892578, 1.0000919103622437, -1.1132265329360962, -0.19640077650547028, -2.504955768585205, 0.007001510821282864, 1.5262320041656494, 0.9996467232704163, -0.5456650853157043, -1.0031049251556396, -1.066352367401123, -0.6113805174827576, 0.7943657040596008, 0.32517996430397034, 3.1826210021972656, -0.6311925649642944, -0.4307652413845062, 0.8156472444534302, 1.6357883214950562, 1.2751963138580322, 0.9539993405342102, 0.3698384761810303, 0.5667832493782043, -0.024836821481585503, 0.32648396492004395, 0.25013092160224915, 0.8385040163993835, 2.1804275512695312, 1.0840685367584229, -0.7854605913162231, 0.03934720158576965, -1.7869011163711548, -0.4017578959465027, 1.3765469789505005, 0.31787189841270447, -0.34751197695732117, -0.3257128596305847, -0.21120230853557587, -2.2948386669158936, -1.0512171983718872, -0.3368300497531891, 0.3347531855106354, -0.3595614433288574, 1.0190750360488892, 0.5373950004577637, 0.13737986981868744], [-0.9231263399124146, 0.1327483355998993, 1.795310616493225, -0.14783936738967896, 0.13974778354167938, 1.291010856628418, -0.24652379751205444, 0.638255774974823, 2.2408759593963623, -0.26506081223487854, 0.0982225239276886, -0.8698874115943909, 0.5049701929092407, 0.21405990421772003, -0.08198966085910797, 0.5607930421829224, 0.13824310898780823, -1.1630752086639404, -0.28953367471694946, 0.23895908892154694, 0.487798810005188, -1.2745460271835327, -0.1809753030538559, -0.9585637450218201, -0.5256351828575134, 1.5496503114700317, -1.4397530555725098, 0.543318510055542, 1.9778193235397339, -0.007769720163196325, 0.06114223599433899, -2.0955283641815186, -0.9665890336036682, 1.9954593181610107, -0.9348350167274475, 0.6569179892539978, 0.005361436866223812, -2.5513460636138916, -1.1070433855056763, 0.13855485618114471, -0.7903352975845337, -0.33482587337493896, 0.872631311416626, -0.5480131506919861, -0.3500474989414215, -0.39993682503700256, -1.1898788213729858, 2.333035707473755, -0.6678741574287415, -1.2106064558029175, -1.7321443557739258, -1.0733177661895752, -1.5265223979949951, -0.37798818945884705, -0.05315925553441048, -0.2722603380680084, -1.973517894744873, 0.982056200504303, -1.7433549165725708, 0.4762922525405884, -0.022869128733873367, 0.07533582299947739, 0.9696061611175537, -0.39890745282173157], [0.5863441824913025, 0.6682419180870056, -0.756504237651825, 0.4805459678173065, 1.0403902530670166, 1.3774340152740479, 0.08995482325553894, -0.022843923419713974, 0.626349925994873, -0.79642254114151, -0.026127096265554428, -0.39994195103645325, 0.8740740418434143, 0.5768263936042786, 1.283353328704834, 0.5601231455802917, -1.1835753917694092, 0.5187358856201172, 1.1327279806137085, 0.09830015897750854, -0.42706120014190674, 1.027600646018982, -0.3469689190387726, -0.07220541685819626, -0.4537011682987213, 1.278699278831482, 0.25983139872550964, 0.09714700281620026, -0.44187602400779724, 0.5509685277938843, -0.5771335959434509, 0.7759630680084229, -0.4189675450325012, 1.401565670967102, -1.9960497617721558, 0.510402262210846, -0.6508823037147522, -0.10819550603628159, -1.0183393955230713, -1.490473747253418, 1.1901775598526, -1.0370359420776367, -0.6336917877197266, -0.2687256932258606, 0.08542191237211227, 1.1918234825134277, 0.29702284932136536, -0.23125137388706207, -0.3509864807128906, -1.497793197631836, 0.7709241509437561, 0.31978172063827515, -1.5684714317321777, -0.9726343154907227, 0.1510365754365921, 0.18732832372188568, 0.21114683151245117, -0.8622468709945679, -2.059926986694336, -0.7259116768836975, -0.7942748069763184, -0.223084956407547, -0.039583250880241394, 0.6244359016418457], [-0.7963591814041138, 0.21432949602603912, 0.44349274039268494, 0.7609111070632935, -0.4492184519767761, 1.6519129276275635, -1.4220541715621948, 0.6623930335044861, 0.3713604211807251, 1.0053343772888184, -0.6263911128044128, 0.8306049108505249, -0.2775183916091919, -0.5248441696166992, -1.8352017402648926, -1.21816885471344, -0.4285070598125458, -1.2028071880340576, 0.1605997532606125, 1.3317095041275024, 0.6608377695083618, 0.07083635777235031, 0.4064638316631317, 1.0537904500961304, 0.8461178541183472, 0.6083332896232605, 1.3887783288955688, 1.025776743888855, -0.8259027600288391, -0.3595447242259979, 0.3299393355846405, 1.909651517868042, 1.626509666442871, 0.026316465809941292, 0.9286234974861145, 1.0949857234954834, 0.8293618559837341, 0.1473013311624527, 1.5577430725097656, -0.1976378709077835, 1.1673110723495483, -1.759131908416748, 0.1964946836233139, 1.829250454902649, -1.2574087381362915, -0.34514519572257996, -0.5232052206993103, -0.22614958882331848, 0.4014662802219391, 0.6190246343612671, -0.07718618959188461, 0.3262615501880646, -0.16003452241420746, -0.15422241389751434, -1.4382247924804688, 2.239870071411133, -2.172452211380005, 0.9868763089179993, 0.32704442739486694, -0.009150871075689793, 1.3194575309753418, 0.02351120114326477, -1.9719173908233643, -0.5114890933036804], [0.8377128839492798, -0.6558237671852112, -0.08626670390367508, -0.2695430815219879, -0.9418209195137024, 1.4295927286148071, 0.6436692476272583, -0.06631851196289062, 0.14893335103988647, -1.1396235227584839, -2.1556951999664307, 0.7738640308380127, -0.04786419868469238, -0.16181789338588715, -0.7835965752601624, 0.24163594841957092, -0.6890867352485657, 0.8288929462432861, 1.1353133916854858, -1.0882327556610107, 0.7681141495704651, 0.9118954539299011, -1.2638949155807495, -0.2910219430923462, -0.9839158654212952, -0.7997452020645142, 0.2537521421909332, -0.21533265709877014, -0.6136355400085449, 0.9572855234146118, -0.09746772050857544, 0.03671502321958542, 0.432192862033844, 1.6155667304992676, -0.681000828742981, 0.05962180718779564, 1.480451226234436, -1.5060688257217407, -1.5591167211532593, -0.943688154220581, 0.3034610152244568, 1.1712843179702759, 0.372811883687973, 0.09939704835414886, 0.31473779678344727, 0.25675851106643677, -0.3990612328052521, -0.6567678451538086, -0.3755253553390503, -0.10634369403123856, 0.3409390449523926, 0.3740171492099762, -1.5795005559921265, -0.42449751496315, 0.20787154138088226, -1.3648279905319214, -1.134289264678955, 0.34184062480926514, -0.9939699769020081, 0.4127578139305115, 0.6089721918106079, 0.5437901616096497, 1.6160497665405273, 0.8632277846336365], [-0.7822979092597961, -0.00019965347019024193, -1.1022157669067383, -0.2756679356098175, -2.241999864578247, 0.2590773403644562, 0.0420626699924469, 0.15818728506565094, 1.0363541841506958, -0.7105397582054138, 0.35309329628944397, -0.050801072269678116, 1.2178175449371338, 1.2718020677566528, -0.4560532867908478, -0.5448055267333984, -0.5371451377868652, 1.0401532649993896, -0.5761920213699341, -0.4304775595664978, -2.1102046966552734, -0.5243953466415405, -1.3194226026535034, -0.8482010960578918, -0.11512846499681473, 1.2069051265716553, 0.3069084584712982, -0.7187287211418152, -1.3682078123092651, 0.22162355482578278, 0.9828304052352905, 1.701535940170288, 1.1219621896743774, -0.11459413915872574, 0.47641226649284363, -1.079233169555664, 0.14567486941814423, 0.21827825903892517, -1.2432093620300293, 0.3381112515926361, 0.2376997172832489, 0.5131478905677795, 0.8843535780906677, -0.1898692548274994, -0.5959381461143494, 1.2042460441589355, 1.0569610595703125, 0.37366795539855957, -1.4296725988388062, -0.08541510254144669, -0.21035711467266083, 0.731130063533783, 0.43229201436042786, -0.5497300624847412, 0.30279430747032166, 0.9565841555595398, -1.0648257732391357, -1.3516689538955688, -0.2964606285095215, -0.11315520852804184, -0.22053712606430054, -1.6082597970962524, -0.0502605102956295, 0.15376010537147522], [-0.08652389049530029, 0.48932087421417236, -1.1303131580352783, -0.9113545417785645, -0.25228074193000793, 1.3151170015335083, -1.0288547277450562, 0.7568178176879883, 0.26256078481674194, 0.14736966788768768, 0.5457403063774109, 0.3555297553539276, -0.23347540199756622, 0.8091462254524231, -0.1873656064271927, -0.29027098417282104, 1.3136215209960938, 0.8169453144073486, 0.34294232726097107, -0.5944045186042786, 1.093785047531128, -0.09817631542682648, -1.579655408859253, -0.3785324692726135, 0.9941073060035706, 1.5802441835403442, 0.583252489566803, -0.810722291469574, -0.943215548992157, -0.7359104156494141, 1.5715677738189697, 0.2436678558588028, 0.18262602388858795, 0.9758417010307312, 2.257765293121338, -0.8236683011054993, 0.5732636451721191, 1.6208875179290771, -1.4860117435455322, -0.5644795298576355, 0.24108943343162537, -2.129225730895996, 0.742813229560852, -0.4207853674888611, -0.43337225914001465, -0.1153814047574997, 0.5399152636528015, 1.283270239830017, -1.2536358833312988, 0.8681589365005493, 1.763535976409912, 0.755517303943634, -0.4811447560787201, 0.04366757720708847, 0.24533867835998535, -1.3391996622085571, -0.026990000158548355, -1.6596112251281738, -1.1180658340454102, -0.8499783873558044, -0.23898620903491974, 0.031048525124788284, -1.0845656394958496, -0.2990613579750061], [-0.2797529101371765, -1.3024084568023682, 0.4528270363807678, 0.39039847254753113, -1.8079360723495483, 0.2684885859489441, 1.0739778280258179, 1.2488185167312622, 0.13397902250289917, -2.645456314086914, 0.7736316919326782, 0.9009148478507996, 0.8059343695640564, 0.5201380252838135, 0.21229921281337738, 0.053149618208408356, -1.3662855625152588, 1.0653769969940186, 0.5889395475387573, -1.000387191772461, 1.1985762119293213, 0.461100697517395, 0.8097476959228516, -0.04842885211110115, -0.8955357670783997, 0.508155345916748, -0.14635562896728516, -0.5007385015487671, -0.7455726265907288, 0.3551531136035919, -0.876010000705719, 0.5917611718177795, 1.7008153200149536, -0.1540662795305252, -0.3196984827518463, 0.13040876388549805, 0.32645219564437866, 1.3572347164154053, -0.5565638542175293, 0.6225594282150269, -0.11613407731056213, 0.34788596630096436, 0.14452041685581207, 0.13396669924259186, -0.6315205097198486, 0.8302322030067444, 0.4976530373096466, 2.4016120433807373, -0.056517329066991806, 1.0057260990142822, -2.089327096939087, -0.07016686350107193, 0.3494070768356323, -0.6353429555892944, 0.5116865634918213, -0.264172226190567, 1.1622475385665894, 0.7773145437240601, -0.1973600685596466, 1.0826972723007202, -0.42352044582366943, 0.5417879223823547, -0.8571192622184753, -0.19481846690177917], [-1.6991989612579346, -0.6985047459602356, 1.3996262550354004, -0.11701416969299316, 0.452081561088562, 0.6052841544151306, -2.206242799758911, 1.384994626045227, -0.5356197953224182, 1.1645475625991821, 0.6473604440689087, -0.8714046478271484, -0.8836778998374939, 0.9083327651023865, -0.13710950314998627, 0.3727671205997467, -0.9005305171012878, 0.5170711278915405, 1.1791340112686157, -1.0556031465530396, -1.5566604137420654, 0.17700161039829254, -0.15808750689029694, -0.8827669024467468, -0.17211835086345673, -1.1389864683151245, 0.8981754779815674, -0.9702785015106201, -0.43640685081481934, 1.4028407335281372, -0.12887847423553467, -0.2631007432937622, 0.8633316159248352, -0.3470783531665802, 0.44899681210517883, 0.9898231029510498, -0.2947680354118347, 1.0675792694091797, 0.1535479724407196, -1.7002729177474976, -1.3335778713226318, 0.7569841742515564, 0.40863120555877686, -1.527555227279663, 0.3463801443576813, 0.6968387365341187, 0.6562075614929199, 1.1392581462860107, -0.9959806799888611, 0.6674296855926514, 0.22422435879707336, 0.6793391108512878, 1.3669703006744385, 1.16256582736969, 2.7108094692230225, 1.1902599334716797, -0.1800660490989685, -1.3001136779785156, 2.0592331886291504, -0.5416805148124695, 0.9857047200202942, -1.272716999053955, 0.6869360208511353, -0.5114026069641113], [-0.0054600914008915424, -0.07773196697235107, -0.7260231971740723, -0.33426710963249207, 0.43333426117897034, 0.8965057730674744, 0.6527929902076721, 1.4592703580856323, 0.653376042842865, 1.9622195959091187, -0.41193366050720215, -1.5728875398635864, -0.7578610181808472, 0.9545093178749084, 0.3549368381500244, -1.2811527252197266, -2.36680269241333, -1.5276681184768677, 0.6473241448402405, 1.1506178379058838, -1.3494817018508911, -0.09792084991931915, -2.961118459701538, 0.5433297753334045, -0.36286023259162903, 0.5655111074447632, 1.0586202144622803, 0.008075417019426823, -0.272723525762558, -1.6718502044677734, 0.05821538716554642, -0.7587642073631287, -0.36834555864334106, 1.5908187627792358, 1.4513468742370605, -1.323730230331421, -0.018489057198166847, -0.8190116882324219, -0.9423270225524902, -0.39484336972236633, -0.11886908859014511, 1.5372750759124756, 0.41866064071655273, -0.20225831866264343, 0.1829291731119156, -1.653760313987732, 1.5634558200836182, 0.7678898572921753, 0.760328471660614, 0.6339424252510071, 0.7323840856552124, 0.9440837502479553, 0.3516855239868164, 0.9556733965873718, 1.58739173412323, -1.3703149557113647, 1.3895952701568604, -0.06523123383522034, 0.5763875842094421, -0.08512264490127563, 0.15583112835884094, 1.307472586631775, 0.6736664175987244, -1.3737999200820923], [-0.2343604862689972, -1.4587806463241577, -0.3874172270298004, 1.2045695781707764, -1.3378572463989258, 1.1099047660827637, -1.6398587226867676, 0.6910713911056519, 0.529773473739624, 0.5685725212097168, 0.39035218954086304, 1.6087955236434937, -0.05019926652312279, -0.3337017297744751, 0.00097211281536147, 0.06222885474562645, 0.2882077693939209, -0.024302633479237556, -0.41357824206352234, 0.9700800180435181, -0.8564630746841431, -0.049951255321502686, 0.5293132066726685, 0.38466203212738037, 0.3404368460178375, 0.6874818801879883, 1.0832200050354004, -0.5052350759506226, 1.8706824779510498, -1.0259642601013184, 0.6395154595375061, 1.7076038122177124, 0.4312427341938019, 0.7383700013160706, -1.4686331748962402, -1.4940510988235474, -0.2540302574634552, 0.13449212908744812, -2.661163330078125, 0.383932888507843, -1.042564034461975, -0.6693246364593506, -0.8915772438049316, -0.5138112306594849, -0.8512340784072876, 1.5941050052642822, -0.32271096110343933, 2.3007380962371826, -0.6864104866981506, 1.1899080276489258, 1.5948307514190674, -0.28357982635498047, 1.2724777460098267, -0.658517062664032, -1.6619889736175537, -2.092024564743042, -2.4264609813690186, 0.681357741355896, -0.430793821811676, 0.7174573540687561, -0.42401114106178284, 0.5326847434043884, 0.2920874357223511, 0.5120336413383484], [0.4275195002555847, 0.5441762208938599, 0.19755618274211884, -0.36817678809165955, 0.7406100630760193, 0.2594413459300995, -2.5039892196655273, -0.8757539391517639, -0.13403798639774323, 0.7297306060791016, 1.2789143323898315, 0.1541014164686203, -0.8220066428184509, -1.0211354494094849, 1.308046579360962, -0.18683895468711853, 1.799043893814087, -0.3344120979309082, -0.817284345626831, -0.3197321593761444, -0.6727505922317505, 0.6333224773406982, -0.974189281463623, -1.1852085590362549, -0.40125319361686707, 1.7473119497299194, 0.7224453091621399, -0.193955659866333, 0.40507498383522034, -0.5398796796798706, 1.789757251739502, -0.7857535481452942, 0.7116915583610535, -0.5079582333564758, -0.6322121024131775, -0.2617684304714203, -1.7603187561035156, -1.0043202638626099, 1.489524483680725, 0.9366860389709473, 0.5490946173667908, 0.1305585354566574, 0.22301773726940155, -0.3459853231906891, 0.3560732305049896, -0.6840226054191589, 0.2557113766670227, -0.06870707869529724, -0.4573306739330292, -1.0697333812713623, -0.2009664624929428, 1.257955551147461, 0.20046496391296387, 0.9316034913063049, 0.9199362993240356, 0.868670642375946, -0.7604854106903076, 1.074782133102417, -1.8948251008987427, -0.4142438471317291, -0.7415949106216431, -1.7666667699813843, -0.7496823072433472, -0.017888303846120834], [-1.0057562589645386, -0.5124408602714539, 0.41906672716140747, 0.12837126851081848, -0.26866018772125244, 0.2152930647134781, 0.46482712030410767, 0.4670945703983307, -0.22481411695480347, 0.21941262483596802, 0.7732247710227966, 0.5392385721206665, 0.7359317541122437, -0.12282146513462067, 0.032291993498802185, -0.49562588334083557, -0.11123775690793991, -0.9126624464988708, 0.191583514213562, 0.3920072615146637, 1.5357993841171265, -0.667995274066925, -0.10742326080799103, 0.8916240334510803, -0.19146548211574554, -0.34313076734542847, 0.948316216468811, 0.4677753150463104, 0.48678138852119446, 1.0731240510940552, -1.5159631967544556, 2.320307493209839, -0.26623082160949707, 1.61078941822052, -0.47360870242118835, -0.3409391939640045, 1.5211647748947144, -0.5886585712432861, -1.3830962181091309, 0.24768072366714478, -1.1933214664459229, -1.5656722784042358, 0.1974005401134491, -1.2079856395721436, -1.5589144229888916, 0.21140193939208984, -0.07063381373882294, 1.2495819330215454, -0.14841073751449585, 1.2505578994750977, 1.1111456155776978, -0.2650420367717743, 0.9942623376846313, -1.300308346748352, 0.35203567147254944, 0.05452892556786537, -1.021034598350525, 1.9159845113754272, 0.8662863969802856, -0.20454953610897064, 0.5326290130615234, 0.11366444081068039, 0.20622693002223969, -0.24098388850688934], [0.6437382698059082, -0.3968586325645447, 0.9812831878662109, 0.3738947808742523, 0.808070182800293, 1.5552549362182617, -0.4729363024234772, -1.941361427307129, 1.3841925859451294, 1.4537203311920166, 0.668874204158783, -1.0883437395095825, 0.39166662096977234, -0.6627930998802185, -0.8926387429237366, 1.4357531070709229, 0.6973920464515686, -1.1875524520874023, 0.7300388813018799, -0.8749223351478577, -1.5167278051376343, -0.28081199526786804, 0.18579547107219696, 0.5354138612747192, 1.531847357749939, -0.262605756521225, -1.7075070142745972, -0.47734689712524414, 0.26892992854118347, 0.5715200901031494, 0.43413034081459045, 0.003907178062945604, -0.5168728828430176, 0.8215704560279846, 0.6824332475662231, -1.2478928565979004, -0.7769986391067505, -0.6069912910461426, 0.06707672774791718, -1.4676398038864136, -0.3972648084163666, 1.1055315732955933, -0.6513838171958923, -0.11740073561668396, 0.02871856838464737, -0.3091847002506256, 0.6393628120422363, -0.3454716205596924, 0.9250698089599609, 0.5702552199363708, -1.0121756792068481, 0.3534873127937317, 0.6431530714035034, -0.12636226415634155, -1.2400658130645752, -0.018765602260828018, 0.2653021812438965, 0.15747441351413727, 0.40633127093315125, 1.392075538635254, -0.4810061454772949, -0.9303329586982727, -0.06878498941659927, 0.8151607513427734], [0.24867230653762817, -0.7429502010345459, 0.18464311957359314, 1.2460649013519287, -0.544629693031311, -0.47258251905441284, 0.28240931034088135, 0.39454567432403564, 1.430715560913086, -1.0696016550064087, -0.9501075744628906, 1.2029917240142822, -0.020665213465690613, -0.574070394039154, -0.2546021044254303, 0.34001874923706055, -0.396275132894516, -1.0629467964172363, 0.8696618676185608, 1.4201081991195679, -0.1792915314435959, 1.6623541116714478, -0.326629102230072, 0.2410910725593567, -0.848804235458374, -0.5317419767379761, 0.3077314794063568, 0.41341808438301086, 0.44927313923835754, -0.043265797197818756, 2.6933884620666504, 1.5225683450698853, 0.6091928482055664, -0.4411829710006714, -0.5758603811264038, -0.6623239517211914, -1.2778879404067993, 0.5807141065597534, -0.8933894038200378, 0.0033287133555859327, 0.8952634930610657, 0.4594128727912903, -0.11423871666193008, 0.6024050712585449, -0.23533275723457336, 0.40521538257598877, -0.5543268918991089, 0.8838136792182922, 0.0009630615822970867, -0.4587155282497406, 1.5127737522125244, 0.8567544221878052, -0.2902221083641052, 0.6592350006103516, -0.5630155205726624, -0.3116253912448883, -0.8968015313148499, 0.18007518351078033, -0.20853346586227417, -2.0398313999176025, -0.5132780075073242, -0.5231155157089233, -0.2664562165737152, 0.018739674240350723], [0.4656219184398651, 0.04519612714648247, -0.017162801697850227, 1.69267737865448, -1.393822431564331, -0.02321852743625641, -0.3010003864765167, -1.056044101715088, 0.8277413845062256, 0.392516165971756, -2.3048014640808105, -0.6554151177406311, -0.6621875762939453, 0.15781661868095398, -0.09228186309337616, 1.5829530954360962, -0.17678649723529816, -0.8964899778366089, -0.08639056980609894, -1.2505892515182495, 0.8854286670684814, 0.42445382475852966, 2.142152786254883, 0.4658346176147461, -0.8536591529846191, -0.24326162040233612, -0.4432543218135834, -0.636038601398468, 0.29531461000442505, 0.41283348202705383, 0.812751293182373, 0.8051827549934387, 0.38609132170677185, -0.10170958936214447, -0.6057151556015015, 0.22436262667179108, -0.7784104347229004, -1.9336544275283813, 0.1624060720205307, -1.122581958770752, -1.850864291191101, 0.5923527479171753, -0.6396908760070801, -1.1674424409866333, 0.07346118241548538, -0.6464457511901855, -0.22322505712509155, -0.05736611410975456, 0.3092348575592041, 1.329189658164978, 1.2532873153686523, 0.48331257700920105, -1.0852328538894653, -0.5393069982528687, 0.6984567642211914, 0.6789851188659668, 0.6656813025474548, 0.28586599230766296, -0.5642472505569458, 1.6917463541030884, -0.6526659727096558, -0.2822670042514801, 1.8088854551315308, -0.7297061085700989], [-0.9699269533157349, 2.2237231731414795, 1.1058365106582642, 0.09786681830883026, 0.21394747495651245, 1.0426121950149536, -0.13501405715942383, -1.1630817651748657, 0.07229426503181458, 0.046536702662706375, -0.61686772108078, 2.7280595302581787, -0.9252272844314575, -0.394539475440979, 1.0240591764450073, 0.8236532211303711, -0.7006608843803406, 1.2745258808135986, -0.6261705756187439, -0.6079955101013184, 0.48265889286994934, 0.6004645228385925, -0.27265676856040955, -1.3510693311691284, 0.42380639910697937, -1.5451788902282715, 0.2739572525024414, 0.2705869674682617, -0.5813933610916138, 0.8425617218017578, -0.9670335650444031, 0.7649297714233398, 0.5515871644020081, 1.5617538690567017, -0.19910721480846405, -2.714956045150757, 0.1150396391749382, -1.6771504878997803, 0.27248966693878174, 1.4609482288360596, -2.203294277191162, -0.15637926757335663, -0.516448974609375, -0.6870083212852478, -0.5722498893737793, 0.8261709213256836, 1.929504632949829, -0.16559365391731262, -1.8879003524780273, 0.48156023025512695, -2.0172383785247803, 0.8352034687995911, -1.18369722366333, -0.09820903837680817, 1.4443678855895996, 0.373616486787796, -1.702122688293457, 0.19931168854236603, -1.3000118732452393, 1.5312161445617676, 0.03279547393321991, 0.6496115922927856, 0.21497072279453278, 0.7669215202331543], [-1.0907304286956787, 1.4421861171722412, 1.0799317359924316, 0.144198477268219, -0.047670941799879074, 1.1150504350662231, 0.5014756917953491, 1.0968406200408936, -0.5560100078582764, -1.068239688873291, -0.5549466609954834, -0.028493573889136314, 0.84916752576828, 0.24716874957084656, 2.0908203125, -1.531586766242981, -0.17111897468566895, -0.059345729649066925, 0.0049280342645943165, -0.45598873496055603, -1.2841377258300781, 1.214760661125183, 0.3540036380290985, -0.813255250453949, -1.0737472772598267, 2.340846300125122, -0.4081658720970154, 0.3964126706123352, -0.05957212299108505, 0.6166152358055115, -2.277111291885376, -0.20383590459823608, 1.084741234779358, 0.1446007788181305, 1.2003313302993774, 0.6865070462226868, 1.311955213546753, -0.5865944027900696, 1.5494108200073242, 0.06896794587373734, 0.394532710313797, -0.1811685413122177, 0.9264959096908569, -0.12107385694980621, 1.353570580482483, -1.3631579875946045, -0.9887033104896545, 1.2642629146575928, -0.4478953182697296, -0.9471524953842163, 0.7145447731018066, -2.508254289627075, -1.2215564250946045, -0.469138503074646, -1.33000910282135, -0.46834930777549744, -0.08538433909416199, 1.2837235927581787, 0.06964577734470367, 2.1339967250823975, -1.6047288179397583, -0.34463343024253845, 0.4850485622882843, -0.2541312277317047], [-0.5016847252845764, 0.2926125228404999, 1.0394518375396729, 1.4430073499679565, -1.4778873920440674, 0.5432031750679016, -0.43136346340179443, -0.4564323127269745, 0.3552142381668091, 0.4319864809513092, 0.585456371307373, 2.1822614669799805, 0.7376757860183716, 0.06496065855026245, -1.1498996019363403, 2.51505970954895, 0.33791491389274597, 0.9830296635627747, -0.6858679056167603, -0.7442514896392822, 0.5090367197990417, 1.362186312675476, 0.4707791209220886, -0.5475622415542603, 0.934743344783783, -1.692478060722351, 0.11515548080205917, -1.0261507034301758, 0.9502502083778381, 0.9911167621612549, -0.5340120792388916, 1.7997492551803589, 1.8128403425216675, 1.6570487022399902, 0.7443188428878784, 0.5278047323226929, -0.19387203454971313, -1.1357494592666626, -1.5262832641601562, -0.12486853450536728, -1.0829784870147705, 1.6485011577606201, 0.6672027111053467, -0.7620477080345154, 0.13157348334789276, 0.5272319912910461, 0.35525983572006226, -1.3723713159561157, -0.46792542934417725, -0.697266161441803, 0.5755947232246399, -0.4410284161567688, 0.27064552903175354, -0.04005148634314537, -1.4342833757400513, 0.24131543934345245, 0.042250074446201324, -0.03979414328932762, 2.4111695289611816, 0.3937622606754303, 0.2585156261920929, 0.260639488697052, 1.37735116481781, 0.1080499142408371], [0.44177040457725525, -0.18230488896369934, 0.7691054344177246, 2.4004485607147217, 0.7476765513420105, 0.40926671028137207, 0.9654182195663452, 0.3419872522354126, 1.3497662544250488, 0.9856892824172974, -0.5819324254989624, -0.6760140657424927, 0.34366899728775024, 0.5650529265403748, -0.4568396508693695, -1.2415839433670044, 0.558102548122406, 0.6933881640434265, -1.0170859098434448, 0.6488401293754578, -0.7552589178085327, -1.4265636205673218, 0.011758432723581791, 0.13506053388118744, -1.135887861251831, 1.8087812662124634, 0.40573635697364807, 1.4998201131820679, -1.1118659973144531, -1.852380394935608, -0.5161446928977966, 0.34815332293510437, 0.1767166256904602, -0.9867947101593018, 0.16628357768058777, 1.8319041728973389, 0.1201794221997261, 0.7421705722808838, -0.04825509339570999, 0.49744588136672974, -0.7662312388420105, 2.147043466567993, -0.5133681297302246, -0.404339998960495, 2.5847909450531006, -0.4468313455581665, -0.5856243371963501, 1.1116480827331543, -0.7276005148887634, 0.8520709276199341, -0.3048740327358246, 1.2106149196624756, -1.5899730920791626, -0.9280822277069092, 2.344237804412842, 2.366856336593628, -0.6325985193252563, -1.1848407983779907, 1.7784581184387207, 1.1469248533248901, -0.19511261582374573, 0.023670747876167297, 0.07199235260486603, 0.79200679063797], [-0.27010083198547363, -1.1163891553878784, -1.2159643173217773, -1.391239881515503, -0.861392617225647, 1.5446618795394897, 1.1182347536087036, -0.9098142385482788, 0.01156553439795971, -0.020888807252049446, 1.0338540077209473, -0.8250328302383423, 0.5094372630119324, -0.6305524706840515, -0.2076554000377655, -0.7952682971954346, 0.8062180280685425, 0.5535686612129211, -0.11630081385374069, -0.2371322065591812, 0.38150742650032043, -0.9424746036529541, 0.7711078524589539, -0.5919147729873657, -0.39130935072898865, 1.17018461227417, -0.7292861342430115, -0.9619506597518921, -0.6499863862991333, 0.7382280826568604, 0.22488242387771606, 0.026268620043992996, -0.31740495562553406, 0.986297070980072, -1.1781187057495117, 0.4202072322368622, -1.8814196586608887, 0.9268186092376709, -0.6157041192054749, 1.711952805519104, -0.1305759698152542, -0.8869404792785645, 1.0011054277420044, -1.4229304790496826, 0.3588590919971466, -1.137628436088562, -0.5055417418479919, -0.8670385479927063, -0.6894775032997131, 0.35224828124046326, 1.9851839542388916, -0.9868174195289612, -0.29035183787345886, -0.2239607721567154, -0.3689011037349701, 1.9367139339447021, 0.09670689702033997, -1.4141247272491455, 0.44964316487312317, -0.42572021484375, 0.008534211665391922, -0.4355063736438751, 1.4226016998291016, 0.40116631984710693], [0.9031803011894226, 0.2905574142932892, 0.6317235231399536, -0.42214760184288025, -0.40179356932640076, -0.3549494445323944, -0.7820225954055786, -0.7092676758766174, 0.6326021552085876, -0.2494029700756073, -0.5351576805114746, 1.1269055604934692, -0.004060619045048952, -0.7932775020599365, 0.9416348338127136, -0.4588279724121094, 0.8628942370414734, 0.28689244389533997, 0.4220103323459625, -2.2085680961608887, -1.0685670375823975, 0.0630895122885704, 0.4203546941280365, 0.24925723671913147, -0.0650831088423729, -0.5367526412010193, 0.3257904052734375, 1.8650672435760498, 0.6402782201766968, 0.862243115901947, 0.13711893558502197, -0.36749711632728577, 1.198426604270935, -1.4865285158157349, -2.47259521484375, 0.969048798084259, 0.28406044840812683, 0.26734456419944763, -2.019955635070801, 1.456905484199524, 0.3451964557170868, -1.2626110315322876, -0.9346247911453247, -0.9057796597480774, 0.15383589267730713, 1.0291903018951416, 1.7614333629608154, -1.6681913137435913, -0.9317632913589478, -0.14166013896465302, 0.46538040041923523, -2.194202184677124, 0.6093316078186035, 0.5305406451225281, -2.535140037536621, 0.17142148315906525, -1.1324045658111572, -1.1166276931762695, -1.2861734628677368, -1.6133134365081787, -0.06837833672761917, 1.6400474309921265, 0.04880799353122711, 0.7889858484268188], [-0.862214982509613, -0.15417304635047913, 0.10663891583681107, -1.7421931028366089, 1.0216976404190063, -0.8846135139465332, 0.9767346382141113, -0.2516731917858124, 0.12329617887735367, 1.9299168586730957, -0.8636897206306458, -0.6149728298187256, 1.1058158874511719, -0.04536283388733864, 0.5391495227813721, 0.8297013640403748, 0.43497392535209656, -0.5149059891700745, -2.054954767227173, 0.2048843502998352, 1.2445775270462036, 0.01942117139697075, 0.8911903500556946, -2.0412325859069824, -1.401505470275879, 0.62662273645401, 1.9588009119033813, 1.043177604675293, -1.245540738105774, -0.9547680616378784, -0.6630780100822449, 0.9981507062911987, -1.819254994392395, 1.2766846418380737, 0.6774494051933289, -0.7310129404067993, -0.87251877784729, 0.23617422580718994, -1.8301044702529907, 2.061217784881592, -0.6644158363342285, -0.8042130470275879, -1.4675453901290894, 0.00010645708971424028, -2.299146890640259, 0.4141773581504822, -0.3531036376953125, -1.3435492515563965, -0.6693550944328308, -0.06860814243555069, -0.031479090452194214, 2.5494894981384277, -0.8417624831199646, 0.7443907856941223, -1.2806828022003174, 0.4186325967311859, 0.05936224013566971, 1.170935034751892, -0.7349494099617004, -0.5809431076049805, 1.5056812763214111, 0.7537164688110352, -0.06934010237455368, 1.1668351888656616], [-0.7039871215820312, 0.37648189067840576, 0.5340087413787842, -0.6519292593002319, -0.029836054891347885, 0.01862199790775776, 0.38873425126075745, -1.5123571157455444, 0.8485959768295288, 0.44629183411598206, 0.8581093549728394, -1.1573362350463867, -1.7851043939590454, -1.7482242584228516, 0.18787778913974762, 0.4383990168571472, 0.013493948616087437, 0.9902733564376831, 0.8651688098907471, -0.3102567195892334, -0.23869171738624573, 1.6969375610351562, -0.7003437280654907, -0.3747303783893585, 0.048764463514089584, 2.491743803024292, 0.3791990578174591, -0.26540103554725647, -0.6126388311386108, -0.9101740717887878, -0.6863547563552856, -0.9325512647628784, 0.3155571520328522, -0.8829671144485474, -0.01677350513637066, 1.3820828199386597, 1.3484349250793457, 1.1320546865463257, 0.30784764885902405, 0.983014702796936, -0.14790791273117065, -0.09401599317789078, 0.3839748501777649, -2.1459457874298096, -0.7242691516876221, 0.7119922041893005, -0.04717720299959183, 0.6644166111946106, -0.613201916217804, -1.1515387296676636, -0.04500270634889603, 0.636936366558075, 0.6025323867797852, 0.20796924829483032, -0.25171560049057007, -1.1763207912445068, 1.0367740392684937, -0.9952327609062195, 0.14190015196800232, 1.1409183740615845, -1.4220390319824219, -0.6887215375900269, -0.27423980832099915, 1.0215295553207397], [-1.146144986152649, -0.841792643070221, 1.5839060544967651, 0.9092833995819092, 1.1832258701324463, -2.166234254837036, 2.430598497390747, -0.6560204029083252, 1.6189651489257812, -0.4588525593280792, -0.4381293058395386, -1.1964353322982788, 0.01811188831925392, 1.6318751573562622, -0.5995141863822937, -0.7552856206893921, 0.437309205532074, 1.2601820230484009, 0.8367273807525635, 1.084390640258789, -0.09185441583395004, -0.44229158759117126, -0.10583055019378662, -0.2123435139656067, -0.7608603239059448, 1.0408614873886108, -0.6726677417755127, -0.445900559425354, -0.21915189921855927, -1.2719979286193848, 1.3128759860992432, 0.6606867909431458, -0.5066842436790466, 0.7941093444824219, 0.1602683663368225, -0.35670122504234314, 0.17387227714061737, 0.20602425932884216, 0.49580544233322144, 0.6062687635421753, 0.5128223896026611, -0.961632490158081, 0.7325345873832703, 1.134198784828186, -2.0513813495635986, 0.74831622838974, 0.07475542277097702, -0.1624307930469513, -0.037664588540792465, 0.7594752311706543, -0.3730948865413666, -2.263522148132324, -1.0227254629135132, -2.1184070110321045, -1.2302125692367554, 0.030309749767184258, -1.0869845151901245, 0.14928004145622253, 0.2806320786476135, -1.203066110610962, -1.1216850280761719, -0.693252444267273, -0.28376469016075134, -0.7373911738395691], [0.7672293782234192, 0.5716387033462524, 1.309434175491333, -0.09999602288007736, -0.3036855161190033, -0.09575311839580536, 0.537916362285614, -2.3842597007751465, 2.0509204864501953, 0.6039140820503235, -0.8770049810409546, -0.5482332706451416, -0.8613461852073669, 1.0579732656478882, -0.1872500330209732, 0.1270635426044464, -0.46086958050727844, 0.9348587989807129, 0.48504236340522766, 0.7110631465911865, 0.1559898853302002, -0.24401552975177765, -1.9098362922668457, 0.6514949798583984, 0.47206994891166687, 0.10934601724147797, 1.7027478218078613, -0.6551849246025085, -0.8171441555023193, 1.9461016654968262, 0.2878129184246063, -1.6504584550857544, -1.3115949630737305, -1.3966153860092163, 0.5198623538017273, -0.28243935108184814, 1.0656758546829224, -0.3931155502796173, 0.4935409724712372, -1.3965801000595093, -0.7212100625038147, -2.51332688331604, -0.7818598747253418, 0.7760485410690308, 1.2957032918930054, -0.3620760142803192, -0.6230543851852417, -1.915493369102478, 1.2033226490020752, 0.9815874099731445, -0.5833122730255127, 0.7496938109397888, 1.7867486476898193, -1.8493173122406006, 1.5214293003082275, 0.14343146979808807, 0.744244396686554, -1.9902493953704834, 0.07584389299154282, 0.1338014155626297, -1.5361603498458862, -0.6585373878479004, 0.3607679605484009, 1.1115890741348267], [1.0267826318740845, -0.2763494551181793, -0.7172073125839233, -0.8657117486000061, 0.24864423274993896, -0.8780989050865173, -0.16594159603118896, 2.664569616317749, 0.043822482228279114, 1.1730445623397827, -0.8107844591140747, 1.61471426486969, 0.0443916991353035, -0.9662603139877319, 0.02111188881099224, -0.027959855273365974, 1.2458561658859253, 1.7713861465454102, 0.47370263934135437, 1.8283913135528564, 0.7519754767417908, -1.1829521656036377, 2.0156989097595215, 0.42910531163215637, -0.6064332127571106, -0.3486231863498688, 0.49021628499031067, 2.001183271408081, 0.4314616024494171, 1.5190712213516235, -0.295432448387146, -0.4572122395038605, 0.008932984434068203, 0.7980241775512695, -0.11707229167222977, -0.23086927831172943, -0.7851408123970032, -1.3118609189987183, 0.5129495859146118, -1.8912805318832397, 0.4766480624675751, -0.0508926697075367, 1.1728620529174805, 0.7197911143302917, -0.0019643385894596577, 1.2661508321762085, 1.005467414855957, 0.06104397401213646, 0.16447153687477112, 0.7552095055580139, 0.6004219055175781, 0.8639386892318726, 0.146851047873497, 0.2765279710292816, 0.47293972969055176, -1.587468147277832, -1.8609164953231812, 0.5488165020942688, 0.46582135558128357, 0.913608729839325, -0.2450912892818451, -1.195685863494873, -0.2001478672027588, -3.0558793544769287], [0.4977497458457947, 0.7147725820541382, 0.4529523551464081, 1.6184942722320557, -0.5955996513366699, -0.2921638786792755, -1.6950994729995728, 2.23459792137146, -0.5175169706344604, -0.6043387055397034, 0.09957724809646606, -1.1923991441726685, -0.3571694493293762, -0.18691697716712952, 0.41306784749031067, 0.6780304312705994, 2.7297797203063965, -2.173180341720581, -0.5779646635055542, 1.3898062705993652, 0.43851256370544434, -1.2682850360870361, -0.8976261019706726, 0.344201922416687, 0.31283292174339294, 0.6064252257347107, 1.7506548166275024, 0.6127011775970459, -0.8300646543502808, -0.5699788928031921, 1.4200265407562256, -0.38678836822509766, 0.09973020106554031, 0.39884600043296814, -0.31629678606987, 0.9802456498146057, -0.5718517899513245, -0.05884929001331329, -1.4033879041671753, 0.4075048267841339, 0.16268347203731537, 0.6838681697845459, 2.0230252742767334, -0.8436854481697083, -0.9356706142425537, -0.2762138545513153, 1.2335315942764282, -0.7576962113380432, 1.340376853942871, 0.8872893452644348, 0.5517129302024841, -0.27581584453582764, -1.188627004623413, -0.9771092534065247, -0.3524476885795593, 2.248945474624634, 0.12330499291419983, 0.2270990014076233, -0.6200374960899353, -0.6620041131973267, 1.6525883674621582, 1.1644560098648071, 0.8165896534919739, -0.022250119596719742], [0.15895038843154907, -1.6339596509933472, 0.06563631445169449, 0.1876671463251114, -1.1900964975357056, 1.0333560705184937, 0.7077005505561829, 0.4988998472690582, 0.9876411557197571, 0.6552664041519165, 1.8082294464111328, -1.0183072090148926, -0.8755322098731995, 2.0260958671569824, 0.7405897378921509, 0.31548064947128296, -2.1160852909088135, 0.4328261911869049, -1.8842700719833374, -0.9306820034980774, 0.03902754932641983, 1.7032369375228882, 0.26026779413223267, -0.6484603881835938, 0.013795692473649979, -0.4433706998825073, 0.20458784699440002, -0.37803059816360474, 1.111507534980774, -0.5484639406204224, -0.2567080557346344, 0.44092079997062683, -0.3945286273956299, 1.261751651763916, -1.3211078643798828, -0.7919215559959412, 0.9313867092132568, -2.0057778358459473, -2.226130247116089, -1.209364414215088, 0.7768092751502991, -0.5482812523841858, -0.5494447350502014, -0.18759265542030334, 0.1886104792356491, -0.6431452631950378, 0.6492911577224731, -1.2534191608428955, 1.3409748077392578, 0.7221779227256775, -0.9371089935302734, -2.2543423175811768, -0.13478749990463257, 0.6035903692245483, 1.3194971084594727, 1.1220954656600952, 1.130868673324585, 0.8607767224311829, -0.40334177017211914, 1.549752950668335, 0.3110368549823761, 0.8045438528060913, 0.05756842717528343, 0.6711851358413696], [-0.05605248734354973, -1.067331075668335, -1.5541255474090576, -2.0119948387145996, 0.7615662813186646, 0.2043873518705368, -0.42559894919395447, 0.5957295298576355, -0.20832626521587372, -0.20229695737361908, 0.8893996477127075, 0.9620974063873291, 0.1718786656856537, 0.2684383690357208, 0.3425246775150299, -0.39431166648864746, 1.1348832845687866, 0.15096615254878998, -0.49426642060279846, -0.5567430257797241, -0.5474464297294617, -1.0952988862991333, -0.8520829081535339, 1.1073777675628662, -0.4747783839702606, 0.5782385468482971, 1.6053056716918945, 1.2186622619628906, -0.7944871187210083, -0.04637773707509041, 0.652920663356781, -1.3843653202056885, 2.5288517475128174, 0.6369801163673401, -2.1374404430389404, 0.8172637820243835, 0.7726354598999023, -0.5445422530174255, -0.43721288442611694, -0.6789580583572388, 0.5245141386985779, -0.6171476244926453, 0.06473538279533386, -0.07106079161167145, -0.5237970948219299, -1.2297862768173218, 0.017789587378501892, -0.7569274306297302, 0.8622934222221375, -0.3945409059524536, -0.41475483775138855, 1.053378939628601, 1.0174198150634766, 0.5496640801429749, -0.21572977304458618, -0.8849292397499084, -0.8446947932243347, 1.402834177017212, -0.7246843576431274, -1.1454650163650513, -0.6064590811729431, -0.14399804174900055, -0.5992773175239563, 0.7625642418861389], [0.08461769670248032, 0.1867748498916626, 0.04374495521187782, 1.5625648498535156, 0.07067199796438217, -0.2126678079366684, -0.180158793926239, -0.23067834973335266, 2.2446305751800537, 0.6126134395599365, 0.18703697621822357, 2.5142245292663574, 1.27415132522583, 3.029796600341797, -1.5119861364364624, -0.17848560214042664, -1.1852439641952515, -1.7689827680587769, -1.219810962677002, 0.0797165110707283, 0.3605385422706604, -0.9061405658721924, -1.4614790678024292, 0.6300907135009766, -0.01813921146094799, 0.886592447757721, 0.0541105754673481, 1.233756184577942, 0.366313636302948, 3.4449689388275146, -0.40031668543815613, 0.28356847167015076, 0.44756031036376953, -0.4800662100315094, -0.15902873873710632, 0.46035537123680115, -0.4619285762310028, -0.7280321717262268, -0.3036239743232727, 0.3676142394542694, -0.18520595133304596, -0.9318280816078186, 0.1423337608575821, 0.8792786598205566, 1.777341604232788, -0.5800665616989136, 0.48007500171661377, 0.0358961783349514, -0.15980663895606995, 0.6277675032615662, 0.2220730185508728, -0.12846173346042633, 0.6260262131690979, 2.582859754562378, -1.5546923875808716, -1.5215821266174316, 1.9767346382141113, -0.663616418838501, -0.5568657517433167, 0.2691033184528351, 0.3151560127735138, 1.4288910627365112, 0.07999978959560394, 0.9162881970405579], [1.3719232082366943, 0.10954930633306503, -0.6337092518806458, -1.7751432657241821, 0.7875116467475891, 1.8697255849838257, -1.5948433876037598, 0.4200346767902374, 0.841323971748352, 1.0876133441925049, 1.8573007583618164, -2.3139054775238037, -0.9606328010559082, 0.2584064304828644, 1.0103927850723267, -0.6167135238647461, -0.7411453127861023, -1.4656946659088135, -1.0010912418365479, -0.023220814764499664, -0.5085135698318481, -0.7892022132873535, -1.1060489416122437, 2.014622926712036, 1.7209208011627197, -0.6013435125350952, -0.47195643186569214, -0.7315070033073425, 0.3892017900943756, -0.3757278323173523, -0.9615028500556946, -0.6619095206260681, -0.5630543231964111, 1.1010822057724, 0.8623034954071045, 0.8848088979721069, -0.5691484808921814, 0.4382542669773102, -0.7131164073944092, -1.9942625761032104, -0.0015461983857676387, 1.349437952041626, -0.9652316570281982, -0.9351294040679932, 2.1445913314819336, -1.6362464427947998, 1.5033386945724487, -1.1599549055099487, -1.3930126428604126, 0.4783586859703064, 0.8104082942008972, 0.9593858122825623, 1.3312441110610962, -1.3405489921569824, -1.3497101068496704, 1.7344962358474731, -0.03092319145798683, -0.39474278688430786, -1.0154979228973389, -0.9374743700027466, -0.22746922075748444, -0.4043584167957306, -0.6042414903640747, -1.3235090970993042], [1.023213267326355, -0.7680618166923523, -0.6492981910705566, -0.6217328906059265, 1.516050934791565, 0.9276461005210876, -0.48173651099205017, -2.0007307529449463, 0.6482036113739014, -0.26699432730674744, 1.2037404775619507, 0.44579777121543884, -1.038169026374817, 1.1623488664627075, 0.9919098615646362, 0.7204775214195251, 0.017997587099671364, -1.1525304317474365, -0.7889324426651001, 0.4393322169780731, 0.03502795100212097, 0.024347875267267227, -0.6731730103492737, -1.8599952459335327, -0.3828166425228119, -0.2603822648525238, 0.14860515296459198, -0.0947355404496193, -1.5154716968536377, -2.9438562393188477, -1.0215394496917725, -0.08400387316942215, 2.7214431762695312, 2.2248098850250244, 1.0561795234680176, -0.8914457559585571, 0.8182579874992371, 1.9131455421447754, 1.4354664087295532, 0.43850380182266235, -0.12874890863895416, -0.9041098356246948, 1.373970866203308, 1.3177837133407593, 0.3691864311695099, -1.1244070529937744, 0.7522739171981812, -1.312816858291626, 0.06587480008602142, 0.0815964788198471, 1.2113032341003418, 1.929429531097412, -0.4966084957122803, -0.3041090965270996, 1.1081448793411255, -0.41414666175842285, -0.30506283044815063, -0.22133761644363403, 0.3366818428039551, 0.9341214895248413, -0.036950599402189255, 0.611061155796051, 0.8501155376434326, -0.3155950903892517], [-0.5566160678863525, 0.24639561772346497, -0.4884241819381714, 1.0150777101516724, -0.26619991660118103, 0.028734171763062477, 0.6256174445152283, 0.5942590832710266, -0.16092167794704437, -0.14441950619220734, -1.2131147384643555, 0.21909482777118683, 0.259653776884079, -1.1665836572647095, -1.0190728902816772, -1.4414429664611816, -1.1264524459838867, 0.8260406851768494, -0.14072087407112122, -1.1046842336654663, 0.8629983067512512, 0.5615594387054443, -0.21098928153514862, -0.6149826049804688, -0.5632635354995728, -0.42028099298477173, 0.6768306493759155, 2.3203330039978027, 1.5920542478561401, 0.2452087551355362, -1.6902157068252563, -0.8925302624702454, 0.27456599473953247, 0.10996758192777634, 0.031131070107221603, 0.8984458446502686, 1.1141979694366455, -1.462086796760559, -0.13260500133037567, 1.0573508739471436, -0.8235760927200317, 0.4067477285861969, -0.9194618463516235, -0.25325480103492737, -1.147668480873108, 0.568725049495697, -1.2959614992141724, -0.5704468488693237, -1.9619333744049072, -0.6739732623100281, 0.913935124874115, 0.01757480949163437, -0.27326852083206177, 0.17257553339004517, 2.5716371536254883, -0.6016486287117004, 1.1058610677719116, -0.9634782075881958, -0.5386059284210205, -0.5894126296043396, -0.8199548125267029, 0.41453245282173157, -0.9801384806632996, -0.13992169499397278], [0.25436538457870483, -0.9413723349571228, 0.009893067181110382, -1.0438721179962158, -1.1340129375457764, 0.23950211703777313, 0.7151504158973694, -0.7648159265518188, -1.8257615566253662, -1.9301228523254395, -0.2812800407409668, 0.390575110912323, 0.7857912182807922, 0.40256017446517944, -0.5555932521820068, 1.102982997894287, 0.2841818928718567, -0.8917856216430664, -0.5811808109283447, -0.47862881422042847, -0.8522215485572815, -0.1139572337269783, -1.1671111583709717, 1.244216799736023, -0.5839969515800476, 2.1050851345062256, -0.30891144275665283, 0.0038955353666096926, -0.8198285102844238, 0.7673056721687317, 0.6388180255889893, 0.08335606008768082, 0.4573359191417694, 0.1891382485628128, 0.36487364768981934, -0.4688757658004761, -1.4662848711013794, -1.2510154247283936, 1.2221511602401733, 0.6098577380180359, -0.8294813632965088, 0.017643261700868607, -1.806246042251587, -0.1551758497953415, -1.3398394584655762, -0.8344013690948486, 0.752097487449646, 1.6776574850082397, -0.4683140516281128, 1.468153953552246, 1.4045993089675903, 1.2146788835525513, -0.8860021829605103, 0.2630739212036133, -0.4643532633781433, 2.1772916316986084, 0.7916223406791687, 1.2039291858673096, 0.0026042417157441378, -0.5154698491096497, 0.17595285177230835, -0.5778343081474304, 0.4926585257053375, -0.573598325252533], [1.6453211307525635, 0.5157492756843567, -0.8526156544685364, -1.1302446126937866, 1.3074488639831543, -0.9485070705413818, -1.3983280658721924, -0.2073880285024643, -1.3006407022476196, -0.8308607935905457, -1.2226516008377075, -1.0724576711654663, -0.7222798466682434, -0.04383936524391174, -1.1207538843154907, 0.6737761497497559, -1.5881491899490356, 1.4646409749984741, -1.1317061185836792, -1.120184063911438, 0.32841727137565613, 0.9603720903396606, 0.677657425403595, -0.8722683191299438, 0.3085317313671112, 1.3555443286895752, 0.36158496141433716, -0.33673256635665894, 0.926928699016571, -0.16785438358783722, -1.575893521308899, -0.7980680465698242, 0.6334568858146667, -0.7522426247596741, 0.276502788066864, 1.247153639793396, 0.051435958594083786, -0.6508930921554565, 0.2432451993227005, -0.637924075126648, 0.037180569022893906, 1.3674163818359375, -0.514394223690033, 0.7785935401916504, -0.461506187915802, 1.2732709646224976, -0.2260737270116806, -2.2597553730010986, -0.4249015152454376, -0.6996352076530457, -0.17163793742656708, 1.7130202054977417, -1.1935697793960571, 1.8256651163101196, 0.4330471456050873, -1.3223942518234253, 0.755468487739563, 0.6373434662818909, 1.325394868850708, -0.045126255601644516, 0.4004928767681122, 0.8977755308151245, 0.7645812630653381, -0.30626851320266724], [-0.975142776966095, -0.47682392597198486, -0.29558366537094116, -0.5692157745361328, -0.5340939164161682, 0.976822018623352, 0.17165349423885345, -0.17709343135356903, 0.7461899518966675, -0.43242138624191284, -1.3242998123168945, -1.0924218893051147, -1.4151933193206787, 0.6421464085578918, 0.43414685130119324, 0.5620198845863342, 1.381309986114502, 0.6869900226593018, -1.4357589483261108, -0.4230347275733948, 0.1136966124176979, 0.5120134949684143, -1.3025497198104858, 0.7238098978996277, -0.46499449014663696, 0.9210795760154724, 1.173376202583313, -0.2917839288711548, -0.2808816432952881, -0.5042142271995544, -1.5817811489105225, 1.0866408348083496, 1.4877864122390747, -0.5648065209388733, -0.43563926219940186, -0.7036901116371155, 0.06365621834993362, 0.7047241926193237, -0.44474655389785767, 0.39946603775024414, -0.6100432276725769, 0.09000037610530853, -1.3534176349639893, 0.08262838423252106, -0.7661716938018799, -1.007428765296936, -0.6451296806335449, 1.1863923072814941, -0.6398952007293701, 2.1785998344421387, 0.1804012954235077, -1.198084831237793, 0.7778699398040771, 0.2381041795015335, 0.8289237022399902, 0.6454869508743286, -1.2227991819381714, 0.029156291857361794, 0.37708571553230286, -2.2896292209625244, 1.9136441946029663, -0.045318830758333206, -0.9241018295288086, -0.1382070630788803], [-0.24431833624839783, 0.7876781225204468, -0.03127728030085564, 0.5203560590744019, -1.4449729919433594, 0.19503429532051086, -0.023154236376285553, 0.4751649796962738, -1.4694887399673462, -0.11087126284837723, -0.3332337439060211, -0.27779364585876465, -1.3132164478302002, -1.238690972328186, 0.34159669280052185, 1.742031216621399, 0.3851553201675415, 0.5798113346099854, 0.49553316831588745, 0.34299665689468384, 0.8398637175559998, -0.007936091162264347, -0.09154895693063736, 0.24441634118556976, -0.34080034494400024, -0.3325497508049011, 0.333446741104126, 2.440645456314087, 1.1530083417892456, 2.722864866256714, 2.6869571208953857, -2.320995807647705, 0.13557572662830353, -0.4564068913459778, 1.3967208862304688, -0.6877890229225159, 0.2349025309085846, 0.8344688415527344, 0.7452002763748169, 0.308927059173584, 1.1528586149215698, 0.3103626072406769, 2.3518378734588623, -0.4770392179489136, -0.35520339012145996, 1.4401156902313232, -0.8330232501029968, 0.11930736154317856, 0.46746698021888733, -0.17695346474647522, 1.234039545059204, 0.7099965214729309, -2.350294351577759, -0.1338532418012619, -0.6755599975585938, 1.6809613704681396, 1.1138995885849, -0.5539519786834717, 0.14995382726192474, 0.6013110280036926, 0.4612376093864441, -0.8282794952392578, 0.20239628851413727, 0.7470220923423767], [-0.3185358941555023, -0.6110926866531372, -1.2052823305130005, -2.3170459270477295, -0.7157375812530518, 0.6095818877220154, 1.308955192565918, 0.5419902205467224, 1.2265686988830566, 1.02118980884552, 1.1562453508377075, -2.5577685832977295, 0.2971593737602234, -0.11598554253578186, -0.7314193844795227, 0.3847085237503052, 1.0450860261917114, 0.9981434941291809, -0.21987265348434448, -0.8572673201560974, -1.4965132474899292, 0.6648012399673462, -0.5452003479003906, -2.0652198791503906, 1.5012470483779907, -1.0057199001312256, -0.9870582818984985, 0.03765817731618881, 0.43154382705688477, -0.013470754958689213, -0.6778449416160583, -0.22743168473243713, -0.9313395023345947, -0.8587668538093567, 1.2195764780044556, -0.3963429629802704, 0.8576797246932983, -0.07368916273117065, -0.38392528891563416, -0.4786258041858673, -0.4897826611995697, -0.16604924201965332, -0.4171660244464874, 0.2184097021818161, -2.3389744758605957, -0.7306035757064819, 1.5762860774993896, 1.0363504886627197, 0.5791863799095154, -0.3173735439777374, -0.1300572156906128, 0.7736724019050598, 2.6871654987335205, 0.18696729838848114, -0.27101171016693115, 0.2593429982662201, 0.4544638693332672, -0.19165107607841492, 1.0130691528320312, -1.6443591117858887, 0.6090371608734131, -1.2150861024856567, -0.019539834931492805, 1.029127597808838], [0.25426316261291504, -1.3136096000671387, 0.025068972259759903, 1.9219609498977661, 0.9385595917701721, 1.0000771284103394, 1.906105637550354, 1.2323452234268188, 1.877943992614746, 0.09160613268613815, 1.134149432182312, 0.9623022675514221, -2.3179593086242676, 0.35208699107170105, -1.5840694904327393, -0.4776802361011505, 1.1040287017822266, 0.07965516299009323, -0.7590468525886536, 0.6723789572715759, -0.900749146938324, -0.7478940486907959, -0.5040662288665771, -0.3391576111316681, -0.2926642894744873, 0.5073447823524475, -0.8678866624832153, 2.0131208896636963, 1.1206910610198975, 1.5571954250335693, -0.26993805170059204, -0.1886589378118515, -0.6526031494140625, -0.8073604702949524, -1.4544168710708618, 1.2171683311462402, -0.6096023321151733, 0.7941296100616455, -0.6304831504821777, -0.17280089855194092, -2.912876605987549, 0.32336434721946716, 0.3863685131072998, 0.6379007697105408, -0.06753281503915787, 0.06999123096466064, -0.48166415095329285, -0.33354678750038147, -0.6467779874801636, -1.8633073568344116, -0.7980251908302307, -0.10530012845993042, -0.5660415887832642, 1.2108793258666992, 0.17401057481765747, -2.3109147548675537, 0.48183053731918335, 0.9353033900260925, -1.067851185798645, 0.05653683468699455, -1.0077557563781738, 0.6153013706207275, -1.8800959587097168, -1.4184197187423706], [-1.5170567035675049, -0.3455483615398407, 0.26252514123916626, 1.2036871910095215, -0.43766194581985474, 0.3511280119419098, -1.3538535833358765, 0.9293372631072998, -0.9811352491378784, -0.42345502972602844, -1.4224259853363037, -1.9146419763565063, 0.5372312068939209, 0.5435413122177124, 0.736566960811615, -0.3772105872631073, 0.3508044183254242, 0.1900123655796051, 0.7945162653923035, 0.6712388396263123, -0.41486939787864685, -0.14100031554698944, -0.5710022449493408, -0.5320872068405151, 0.289024293422699, -0.6684401631355286, 1.5947320461273193, 0.9313185214996338, 0.4428105056285858, -0.7930368781089783, 0.48223957419395447, 0.8917911648750305, 1.6147476434707642, -0.15686941146850586, -0.17905540764331818, -1.0467321872711182, 0.29242998361587524, -0.39357897639274597, -1.4737186431884766, -1.3831034898757935, -0.4656105637550354, -0.1272083967924118, -1.4732609987258911, 0.5375564694404602, 0.5342682600021362, 0.40882059931755066, -0.9928600192070007, -0.23238928616046906, -0.4113353192806244, 0.5863443613052368, 0.2928149402141571, 2.0451908111572266, 0.8379961252212524, 0.4899693727493286, 1.4732691049575806, -0.44310811161994934, -0.11239489167928696, -0.3040619492530823, 1.1248531341552734, 0.20798733830451965, 1.3274197578430176, -0.7602980732917786, -1.1005488634109497, 0.11152151226997375], [-1.4789938926696777, -1.2130188941955566, -1.6187642812728882, 0.24552981555461884, 1.1829662322998047, 1.7399989366531372, 0.11180175840854645, 0.2540460526943207, 0.33671462535858154, -0.07713919878005981, 0.285905659198761, 0.4955456554889679, -0.1337997317314148, -1.050469160079956, 0.7873937487602234, -0.265998512506485, -0.30115118622779846, 0.07968631386756897, 1.670622706413269, -2.0395021438598633, 0.8668546676635742, 0.3466612994670868, -0.48294079303741455, 1.797063946723938, -2.363973617553711, -0.44463998079299927, -0.7574038505554199, 0.6262075901031494, 0.11252147704362869, 0.45207399129867554, 1.8300306797027588, 1.0030372142791748, 0.1491713970899582, -1.0233912467956543, 1.0521292686462402, 1.4173688888549805, 0.2796329855918884, 0.0074639711529016495, 0.6167610883712769, -2.68526029586792, 0.4387197494506836, 0.9631824493408203, 1.4609493017196655, -0.7836334705352783, 0.4851231575012207, -1.6605913639068604, 0.037594426423311234, 1.4234764575958252, -0.5422018766403198, -2.397899866104126, -0.9477093815803528, -0.27842146158218384, -0.8144868612289429, 0.701074481010437, -0.2501070201396942, -0.37265947461128235, 0.384026437997818, 0.5433165431022644, -0.26773110032081604, -1.7562998533248901, 0.25923070311546326, 0.3392148017883301, 0.6438816785812378, 0.3341907858848572], [-0.9235219955444336, -1.449951410293579, -1.373583197593689, 0.8951112031936646, -0.7546190619468689, -1.3315744400024414, -1.5044903755187988, 0.29232290387153625, -0.5924917459487915, 0.42123305797576904, 0.3448706567287445, 1.4048992395401, 1.2117124795913696, 0.4883557856082916, -0.9887705445289612, -0.1430351734161377, -0.22535543143749237, 0.270267128944397, -0.7934629321098328, -1.4709874391555786, 2.2568254470825195, 0.9275382161140442, 0.028530461713671684, -0.6080016493797302, 0.26386040449142456, -0.426013708114624, -1.5264345407485962, 0.26573723554611206, -1.011633038520813, 0.3231847286224365, 1.0000224113464355, -0.6081262826919556, 0.29855772852897644, -0.3927266299724579, 0.9560074210166931, -1.7908153533935547, -1.0797892808914185, -1.6706302165985107, 1.7835232019424438, -0.9642882347106934, 0.08043123036623001, -1.2150918245315552, -0.24764853715896606, 1.1623135805130005, -0.7820099592208862, -0.12996742129325867, -0.3259788751602173, 1.0043092966079712, 0.3987831175327301, 0.12795747816562653, -0.7395279407501221, 0.35752370953559875, -0.8627287149429321, 0.7347086668014526, 1.8083661794662476, -0.7043658494949341, -0.49521884322166443, -0.899326503276825, 0.9554305672645569, -1.3317707777023315, -0.9291773438453674, 0.4861569106578827, 0.3831041157245636, 2.5614161491394043], [0.4503365159034729, 0.7578110694885254, -0.7199013233184814, 1.5469951629638672, -2.7033653259277344, -1.1748409271240234, 0.49913036823272705, -0.40380799770355225, -0.1583285927772522, -2.7226805686950684, 0.7350839376449585, -0.2593510150909424, 2.0743560791015625, -1.9923303127288818, -0.814608633518219, 0.45308393239974976, -1.3958899974822998, -0.3602306842803955, -0.3624800443649292, -0.6742128133773804, -0.07378268986940384, -0.4492454528808594, 1.0031999349594116, 1.052901029586792, 0.3372490108013153, -1.834202527999878, -1.38456392288208, 0.09050643444061279, -0.2881263792514801, -1.951164960861206, 0.7815724611282349, 1.7037407159805298, 0.3105834722518921, -1.0087449550628662, -0.3326132893562317, 1.254417061805725, -1.1768993139266968, -0.5505125522613525, -0.407388836145401, -0.3311424255371094, 0.6413866877555847, -1.697950839996338, 0.013157472014427185, 2.1200664043426514, -0.2079111486673355, 1.7284835577011108, -0.21816590428352356, 0.15367725491523743, -0.6066029667854309, 0.2657315731048584, 0.029694722965359688, 0.7149916887283325, -0.23971213400363922, -0.18623624742031097, -0.6590573787689209, -0.5217934846878052, -0.43329477310180664, 1.0313844680786133, 0.1053069457411766, 0.08699102699756622, 1.0672707557678223, -0.17523887753486633, -1.3295354843139648, 0.7979410886764526], [-0.5076965093612671, -1.0225969552993774, -1.0377589464187622, -0.40431472659111023, -2.4584929943084717, -1.9105676412582397, -0.716530978679657, -0.17074576020240784, 0.19212855398654938, -0.22864685952663422, 0.5484001636505127, -1.6657559871673584, -1.3961694240570068, 0.27690455317497253, -0.3505256772041321, -0.08010493218898773, -0.6381319165229797, -2.4196460247039795, -1.7377146482467651, -0.5323294401168823, -0.6209127902984619, 0.48986440896987915, 0.050693709403276443, -0.046148642897605896, -0.3750869631767273, 0.43772512674331665, 0.771037220954895, 0.16621245443820953, -0.5759067535400391, 1.2996255159378052, 0.24583591520786285, 0.49309587478637695, -0.04728914052248001, -1.700299620628357, -0.9721665382385254, 0.5639810562133789, 0.8773489594459534, -0.28627198934555054, 1.1502901315689087, 1.5107790231704712, 0.20654955506324768, -0.13712605834007263, -1.3658621311187744, -1.0337910652160645, -1.4452571868896484, 1.0866670608520508, -0.08705684542655945, -1.5402132272720337, -1.6740002632141113, 0.23000496625900269, -1.938393235206604, -0.2011302411556244, -0.1800573766231537, 0.07935144007205963, 0.015282567590475082, -0.611467719078064, -1.296998143196106, -1.4181374311447144, 0.4377562403678894, 0.6123618483543396, -0.2159365862607956, 1.3568326234817505, 0.21185074746608734, 0.8098621368408203], [1.1768943071365356, 0.89601731300354, -0.7949422597885132, -0.11900512129068375, 0.3198169767856598, -0.6175106763839722, 0.5923174023628235, -0.01620309427380562, -0.7420880198478699, -1.879623293876648, -1.148729681968689, 0.5611519813537598, 0.120191790163517, 0.18781904876232147, 0.544828474521637, 0.2432478368282318, 0.3756938576698303, 0.427835077047348, 0.6166466474533081, -0.42935699224472046, 0.8942025899887085, 0.328519731760025, 0.07781926542520523, -0.75666344165802, -1.2283360958099365, -0.6840527653694153, 1.438614010810852, -0.5351960062980652, 0.608866274356842, 0.042810533195734024, 0.04371420294046402, 0.9650607705116272, 0.07414088398218155, 0.6266258955001831, -1.2339887619018555, -0.6784319877624512, -0.39472031593322754, 0.9721308350563049, -0.7266201972961426, -0.2762193977832794, 1.6083279848098755, -1.4500950574874878, 0.10000362247228622, 0.07725977152585983, -1.0636169910430908, 0.26634112000465393, -1.6540263891220093, -1.8464958667755127, 0.6198683977127075, 1.6373602151870728, -0.8783626556396484, -0.40952202677726746, -0.7830825448036194, -0.28192922472953796, -0.34471917152404785, -1.2682894468307495, -0.3459891080856323, -1.2107419967651367, -1.2531605958938599, -0.44215795397758484, 0.20489610731601715, 0.3456902503967285, -0.5650478601455688, -1.994992733001709], [0.08871395885944366, -1.9403115510940552, -1.5461106300354004, -1.001049280166626, -2.4851441383361816, -1.4621026515960693, 0.8340656161308289, 0.9364407658576965, 0.7136299014091492, -0.16713592410087585, -0.5811075568199158, 1.4957752227783203, 0.16793417930603027, 1.488019585609436, 0.055312588810920715, 1.0051015615463257, -0.2700294852256775, 0.5859021544456482, -0.20834201574325562, -2.1690640449523926, 0.3927193880081177, -0.2743390202522278, -2.125951051712036, -0.08958642929792404, -0.2789499759674072, 1.0851731300354004, 0.8879632353782654, 1.0090699195861816, -0.14823684096336365, 0.09693149477243423, -0.22705915570259094, -1.3323808908462524, 0.17406831681728363, 0.5647143125534058, -0.011560031212866306, -1.0031343698501587, 0.1355784833431244, 0.49045321345329285, 0.771064043045044, 1.3326247930526733, -1.6248176097869873, -0.26687511801719666, 0.14530815184116364, 2.1771488189697266, -0.9718533754348755, 0.00743063585832715, -0.11612655967473984, 0.6220756769180298, 0.8873447179794312, 0.9811245799064636, -0.9589399099349976, -1.7548859119415283, -0.8085068464279175, -0.7799132466316223, 0.08916240930557251, 0.7220706939697266, -0.06470215320587158, -0.6122384071350098, 0.796816885471344, 0.06255602836608887, 1.7021467685699463, 0.483369380235672, 1.6599386930465698, -1.9234768152236938], [-0.8614194393157959, -0.8533397912979126, 0.5346737504005432, 1.4741994142532349, 0.650617241859436, 1.1281707286834717, -0.37468042969703674, -1.0469406843185425, 0.7258151173591614, 0.3969469666481018, 1.2086162567138672, 0.5007101893424988, -0.5179367065429688, 0.2987672686576843, 0.35811054706573486, -0.347826212644577, 2.646601915359497, -2.036363363265991, -0.36087819933891296, 0.3753748834133148, -1.9147065877914429, -0.741047739982605, -0.11602192372083664, -0.5489051342010498, -0.5677554607391357, -0.4555968940258026, -0.7692191004753113, -0.7288241982460022, -1.338743805885315, 0.4860494136810303, 0.5465781688690186, -0.5345398187637329, 1.7113957405090332, -0.2065785974264145, -0.5937684774398804, -1.2898626327514648, 0.8500616550445557, -1.0765388011932373, -0.719649612903595, -0.3355582654476166, 0.46866878867149353, -0.04783696308732033, -1.8133068084716797, -0.35596609115600586, 0.6932576894760132, -0.9950674772262573, 0.3133314251899719, 0.5768908858299255, -0.530876874923706, -1.0231997966766357, -1.1109304428100586, 0.5614500045776367, -1.2553906440734863, 0.8128634691238403, 1.1143227815628052, 0.9836621880531311, -0.8951687812805176, -1.3487604856491089, 0.12855489552021027, 0.13093645870685577, -0.9159497618675232, 0.04325097054243088, -0.7109955549240112, -0.8262313008308411], [-0.004958572797477245, -0.2859843969345093, -1.737013339996338, -0.7182399034500122, 0.6534210443496704, 1.084572434425354, 0.035294923931360245, -0.5661476850509644, 2.1967697143554688, -0.8300882577896118, 0.43617457151412964, 0.6321403980255127, 0.28746333718299866, 2.389399528503418, 0.052689120173454285, 0.6987606287002563, 0.18846897780895233, -1.242729663848877, 1.2429883480072021, -0.08708738535642624, 0.8466677665710449, -0.906089723110199, 0.06811999529600143, 0.3706566095352173, -0.6830275058746338, -0.37237992882728577, -1.212914228439331, 0.20713278651237488, -1.401173710823059, 0.42459049820899963, 1.3253382444381714, -0.040175046771764755, -2.4042794704437256, -0.8439716696739197, -0.7784008979797363, -0.4556007385253906, 0.3500126302242279, 1.1030536890029907, 0.17206890881061554, 1.4759447574615479, -0.6440750956535339, 0.10613055527210236, -0.2524683177471161, 0.12425088882446289, 0.062041960656642914, 0.776280403137207, 0.5798834562301636, -1.055206537246704, 2.7808516025543213, -0.6760158538818359, 0.29892411828041077, 0.735724925994873, -0.14710989594459534, -0.9950563311576843, -0.620623767375946, -0.8666232228279114, 1.9740453958511353, 0.19227533042430878, 0.4080299437046051, -0.10969250649213791, 1.2425285577774048, -0.27877727150917053, 1.3205957412719727, 0.14903493225574493], [-1.1184293031692505, -1.560206651687622, 0.042628444731235504, -1.112892508506775, 0.7567546367645264, -1.7090181112289429, 0.1546754688024521, 2.411571502685547, 0.3811759054660797, -1.0783910751342773, 1.2824571132659912, 0.5187180042266846, 0.023849448189139366, -2.0646989345550537, -0.9929487705230713, 0.9501610994338989, -0.9197449684143066, 1.0305148363113403, -0.6084526777267456, 0.5526644587516785, 0.44705840945243835, -0.06621576100587845, -2.050654172897339, -0.036377836018800735, 0.3321983814239502, -0.1550872027873993, 1.4755284786224365, -0.6522940993309021, 0.20883367955684662, 0.6099640130996704, 0.5701620578765869, -0.38036924600601196, 1.187333345413208, -0.3587811291217804, -0.581708550453186, -0.45730289816856384, -0.09482525289058685, 2.0138349533081055, -0.1452091485261917, 0.05045381188392639, -0.30553239583969116, -0.1317901462316513, -0.5546010136604309, 1.840786337852478, -2.869572877883911, 2.1752514839172363, 1.0591763257980347, -0.7468174695968628, 0.8400652408599854, -1.0067285299301147, -1.434141755104065, -0.7163805961608887, 0.14534741640090942, 1.139871597290039, -1.0022205114364624, 1.5800151824951172, -0.0799628347158432, -0.1796943098306656, -0.5708964467048645, 0.29910987615585327, 1.496694803237915, -0.8062490820884705, 0.3469967544078827, -0.20597778260707855], [0.31499072909355164, -0.7029964923858643, 0.21982309222221375, 0.05616520345211029, -0.8258159756660461, 0.16556376218795776, -0.9339149594306946, 0.07109902054071426, 0.8210410475730896, 0.4537431299686432, -1.3909635543823242, 1.55817711353302, -1.0661978721618652, -0.15458735823631287, -0.23813720047473907, 0.8480398058891296, -1.2261818647384644, -0.23162120580673218, 0.4545938968658447, 0.18851123750209808, 0.30034157633781433, -1.3694119453430176, -0.04227152466773987, 0.33956626057624817, 0.496646910905838, 0.5347588658332825, -2.3259518146514893, 1.2125234603881836, 1.2481838464736938, 0.017628764733672142, -2.130323648452759, 0.6769553422927856, 0.9404131770133972, -0.03696485981345177, -0.6714533567428589, -0.7644930481910706, 0.041817739605903625, -0.4932761788368225, -0.34928107261657715, -2.594619035720825, 0.9256331920623779, -0.6395750045776367, -0.030984798446297646, -1.7289941310882568, -0.4849337339401245, -1.6801754236221313, 1.1670258045196533, 1.0612083673477173, 0.04818372428417206, 0.9825647473335266, 1.346290946006775, 1.479826807975769, -0.8501983880996704, 1.5746103525161743, -0.7404754757881165, -0.3995855748653412, -0.5242056846618652, -0.6619752645492554, 0.17635664343833923, -1.1273236274719238, 1.0601814985275269, -2.1447598934173584, -0.8193466663360596, 0.272317498922348], [-0.9369820356369019, -0.42860302329063416, -0.14329662919044495, -0.6242929697036743, -0.03794979676604271, 0.7815106511116028, -0.09004715830087662, -0.5240282416343689, 1.2768628597259521, -0.6991449594497681, -0.040577568113803864, 0.254607230424881, 0.02899031899869442, 0.9854360818862915, 0.06176823005080223, -0.38908299803733826, -0.11738473176956177, -0.00743140559643507, -0.014879673719406128, 0.05209076777100563, -0.8010891079902649, -0.3324242830276489, 1.3865853548049927, 0.7852848768234253, 1.261244535446167, -2.6151950359344482, -1.6078764200210571, -0.2593151032924652, 1.4863762855529785, 0.38355371356010437, 0.01080259308218956, -0.43746188282966614, 1.3603019714355469, 0.5280264616012573, -0.40303534269332886, 0.5781606435775757, -2.020850419998169, -0.9443742632865906, 0.31446072459220886, -0.33018070459365845, 0.5078203678131104, -2.5814552307128906, 0.90169358253479, 2.4654088020324707, 1.6684292554855347, -0.3610239326953888, -0.5869687795639038, 1.0855120420455933, -1.0934704542160034, -1.4570449590682983, 0.8755012154579163, -0.20430880784988403, 1.4219456911087036, -1.4512337446212769, -1.5959981679916382, -0.9658953547477722, -1.0319877862930298, -0.12447638809680939, 1.213479995727539, 0.7781542539596558, 1.044486165046692, -0.9691498875617981, -1.583816647529602, -0.2674667537212372], [0.51972496509552, -0.06747402995824814, 0.9048991203308105, 1.6296985149383545, -0.7694389224052429, -0.8340669274330139, -0.013978018425405025, 1.0050474405288696, -0.7386162877082825, 0.32571354508399963, -0.8392813205718994, 0.4904431998729706, 1.3914093971252441, 0.7588182687759399, -0.2975056767463684, -0.6435391902923584, 0.07975810766220093, -1.176619052886963, 0.8205611705780029, -1.6598135232925415, 1.781488060951233, 0.05991506949067116, 0.454215943813324, -1.3310195207595825, 0.3075392246246338, 2.386455774307251, -1.0133776664733887, 0.350726842880249, -0.5184630751609802, -0.10566773265600204, -0.8643447160720825, 0.8922336101531982, -0.7855260968208313, -0.8940674662590027, -0.17695893347263336, -0.3936549425125122, 0.06427615880966187, 0.6640430092811584, -1.2203439474105835, -1.4596091508865356, -0.11792704463005066, 0.5448266267776489, -2.2479090690612793, 1.0085684061050415, -0.7165004014968872, 1.5009076595306396, 2.7886977195739746, -0.9318915009498596, 0.400328665971756, 0.3031049966812134, 0.557735025882721, 0.5092570781707764, 0.8782677054405212, -0.6393930315971375, -0.6642986536026001, -0.5437269806861877, -0.26081058382987976, 1.2077628374099731, -0.34145811200141907, 1.4341001510620117, 0.376249223947525, -1.133357286453247, 0.30590343475341797, 0.5569286942481995], [1.486342191696167, -1.0772026777267456, 0.6420044302940369, 0.5469832420349121, -1.5686590671539307, -1.8425018787384033, 1.6106890439987183, -0.7921568751335144, 0.54594486951828, 0.18923598527908325, -0.0013306820765137672, 0.5991889238357544, -1.875194787979126, -0.13749098777770996, 0.6374475955963135, 1.405908465385437, -1.1643099784851074, -0.14305716753005981, 1.6069200038909912, 0.13254058361053467, 0.7524860501289368, 0.6557891964912415, -0.15615151822566986, -0.6296644806861877, 0.07762463390827179, -1.6728527545928955, 0.2763104736804962, 1.2317132949829102, 0.9229505658149719, -0.07006386667490005, -1.581188440322876, 0.8395592570304871, -1.178366780281067, -0.7404043674468994, -1.3436412811279297, -0.07247799634933472, -2.6344428062438965, -0.6465558409690857, -1.7498615980148315, -0.32751065492630005, 0.7204486131668091, 2.1029279232025146, -0.6617052555084229, -0.49364688992500305, -0.6086206436157227, 0.7147086262702942, 0.3302736282348633, 1.937389612197876, -0.6896733045578003, 0.5951709151268005, -1.2660340070724487, 0.44097235798835754, -1.2170292139053345, 1.4896165132522583, 0.7264021039009094, -0.8174472451210022, 1.2973073720932007, -1.1557154655456543, 0.9631496667861938, 1.3408864736557007, -1.9515957832336426, 0.8127450346946716, -1.180509328842163, -0.07283312827348709], [-0.8870328068733215, -0.03334512934088707, -0.25134989619255066, -0.07886100560426712, 1.8220762014389038, -0.0629689171910286, 1.3039710521697998, 1.252563238143921, 0.9896693229675293, -0.5995385050773621, 0.16137394309043884, -0.6724859476089478, -0.15686649084091187, -0.6686698794364929, 0.5082272887229919, 0.04576453939080238, 1.1117792129516602, -0.09503228217363358, 0.5025557279586792, -1.311859369277954, 0.4339880049228668, -0.39302897453308105, 1.8236119747161865, 0.6620873808860779, -0.9371363520622253, 0.6144155859947205, 1.3801008462905884, 0.8625509142875671, 0.4248960018157959, -1.1115922927856445, -2.1488773822784424, 0.5375092625617981, 0.2685067355632782, -2.256117105484009, 1.7014118432998657, 1.2764266729354858, -1.2680201530456543, -0.15004809200763702, 0.5478659272193909, 0.7883177399635315, 0.42676112055778503, -0.9287323951721191, -0.8908993601799011, -1.9119757413864136, -1.1490380764007568, -0.22876474261283875, -1.1996351480484009, 0.37661799788475037, 0.23774927854537964, -0.10783722251653671, 0.5556532144546509, -0.2467823624610901, -0.8738890290260315, 0.1752418577671051, -0.08080117404460907, 0.2527991831302643, 1.3620659112930298, -1.392152190208435, -0.020284950733184814, -0.34699541330337524, -2.0721044540405273, 0.5888810157775879, 1.5624210834503174, -0.6116085052490234], [-1.2435047626495361, 0.2016219198703766, 0.053592074662446976, -1.3525872230529785, 0.9836011528968811, -0.16695238649845123, -0.184430330991745, -0.5089863538742065, -1.3260128498077393, 2.144077777862549, 0.9298450350761414, -0.7435618042945862, -1.3728374242782593, 0.49079620838165283, 0.10312756896018982, -1.5101211071014404, -0.1188337579369545, 3.0817785263061523, -0.5159866809844971, -0.11830301582813263, -0.3790956735610962, -0.2863650619983673, 0.5009700059890747, 0.5354381203651428, 0.9608248472213745, -1.2449157238006592, -0.8225693106651306, 1.247624397277832, 0.6844233870506287, -0.4827863872051239, 1.9244540929794312, 0.06384801864624023, -3.131122589111328, -0.1820763349533081, 0.3856700658798218, 0.42556488513946533, 1.1454904079437256, -1.6564791202545166, -0.32404422760009766, 0.5432440638542175, -0.5977649688720703, 0.17129839956760406, -0.013948128558695316, 1.539656639099121, 0.3118208050727844, 0.5411534905433655, -0.8862708210945129, 0.22702409327030182, -0.502555787563324, -1.6230438947677612, -0.159807026386261, 0.44453710317611694, -0.8250445127487183, 0.43331170082092285, 0.32175004482269287, 0.8872425556182861, -0.07205215096473694, -0.5668705701828003, 1.4843993186950684, 0.8119195103645325, 2.0813779830932617, 0.5847955346107483, 0.06365738809108734, 1.6065454483032227], [-0.5059043765068054, 0.7812983989715576, 0.14116378128528595, 1.9772117137908936, 0.6559857726097107, -0.4301539361476898, -0.15324386954307556, -0.842572033405304, 0.14927321672439575, -0.3781743347644806, -0.5928841829299927, -0.18554392457008362, -0.47035327553749084, 1.0097600221633911, 1.3062738180160522, 0.2811563014984131, 0.19633328914642334, 0.36301112174987793, 1.4742122888565063, 0.4306134879589081, -0.6207640171051025, -0.5440356731414795, -0.07249017059803009, -0.4302816390991211, -0.547721266746521, 1.0007683038711548, 0.19485460221767426, -1.1139748096466064, 0.9220594167709351, -0.28341442346572876, 0.12312334030866623, -0.4585016071796417, -0.39225420355796814, -0.13368922472000122, -0.7104098796844482, 0.6372198462486267, 0.9194296002388, -0.5991479158401489, 0.7956467270851135, 0.051347099244594574, 0.67913419008255, -0.5355928540229797, -0.9879317879676819, 0.13860732316970825, 0.9352545142173767, -0.7359216809272766, -1.2387949228286743, 1.459182620048523, 0.960642397403717, -0.5605249404907227, 0.5797197222709656, 0.4558694660663605, -1.2479122877120972, 3.040034532546997, 0.7391201257705688, -1.7763019800186157, 1.4450693130493164, -0.7770900130271912, 1.3597253561019897, -0.7332717776298523, 1.929802656173706, 2.359027147293091, 1.3810169696807861, 0.4220829904079437], [0.0710953027009964, 0.8779181838035583, -0.4504930078983307, 0.2209276258945465, 0.7028070688247681, 0.5631250739097595, 0.15646401047706604, 1.771073341369629, 0.11765008419752121, 1.0696569681167603, -0.40451866388320923, -0.8165711164474487, 0.05531097203493118, 1.647904396057129, -0.4774622321128845, 0.24680401384830475, 0.6382026076316833, -1.2031636238098145, -0.8161501884460449, 0.8755481243133545, -1.1031231880187988, -0.6461915969848633, -1.0074856281280518, -0.5937174558639526, 1.968760371208191, -1.2073949575424194, 0.48655861616134644, 1.7557432651519775, 1.1315902471542358, -0.7089478373527527, -0.9735523462295532, -0.6093533635139465, -0.006397496908903122, 0.7551270127296448, -0.9224350452423096, 3.2552382946014404, -1.0630403757095337, -1.401810646057129, 2.0594005584716797, 0.5672222971916199, -0.2346297949552536, 0.2354678213596344, 1.4131526947021484, 1.4450063705444336, -0.6454683542251587, 1.3657002449035645, -1.1161253452301025, 0.6256524324417114, 0.7146114706993103, -0.586829423904419, 0.7736184597015381, -0.16777987778186798, -0.6441552639007568, 0.6098979711532593, -0.5254100561141968, 1.9128599166870117, 0.07084871828556061, 0.8456358313560486, 1.724983811378479, -0.1614047884941101, 0.4324939250946045, -0.8530619144439697, 0.024373801425099373, 0.5813893675804138], [1.214276671409607, -0.7004987001419067, -0.2378516048192978, -0.20312318205833435, 1.3501849174499512, -0.7197098135948181, -1.764075517654419, -0.9274643063545227, 0.831158459186554, -0.41612571477890015, -0.04244454205036163, 0.23614221811294556, -0.8526136875152588, -2.1042094230651855, 0.6595498323440552, -0.8013104796409607, 1.6927614212036133, 0.1787523627281189, -0.4752722680568695, -1.385792851448059, 0.14636006951332092, -0.32927316427230835, 1.3826590776443481, 0.1350097805261612, -0.5881709456443787, 0.5203195810317993, -1.5579553842544556, 0.40714123845100403, -0.308937132358551, -0.2985544502735138, 1.0047414302825928, 1.305962324142456, -0.42267706990242004, -0.8632915616035461, -1.3111984729766846, -1.0844476222991943, -0.7922406196594238, -0.8287424445152283, 1.8350098133087158, -0.3270965814590454, -0.6051642298698425, 0.9804100394248962, 0.46433916687965393, 0.6273263096809387, -0.5977240800857544, -1.0568008422851562, -0.9089654684066772, 0.6210392713546753, 2.1419618129730225, -0.24500364065170288, 0.2025201916694641, 0.8908706307411194, 0.4320105314254761, -0.35808587074279785, -0.5068110227584839, 0.1969635933637619, 0.7717306017875671, 0.31935638189315796, 1.805128812789917, -1.8759093284606934, 2.5167155265808105, 0.43230992555618286, 0.8560741543769836, 1.4027308225631714], [0.577523410320282, 0.3528951108455658, 0.6985763907432556, 0.1539694368839264, -0.5882657766342163, 1.277815341949463, -0.5826116800308228, 0.8014246821403503, 0.14556171000003815, -2.1037096977233887, -0.3899764120578766, -0.9852901101112366, -0.6735789179801941, 0.8519015908241272, -0.4955926239490509, -1.5147714614868164, -0.7337287664413452, -1.1375178098678589, -1.2215710878372192, -0.1473628580570221, -0.20961381494998932, -1.3821660280227661, -0.5525522232055664, -0.9537019729614258, -0.6238829493522644, 0.7371358871459961, 0.9915732741355896, 0.4141341745853424, 0.4328892230987549, -1.0863933563232422, -0.2915746867656708, 1.276904821395874, 0.09930594265460968, 1.2613364458084106, -0.5102077126502991, 2.0367536544799805, -2.1298716068267822, 0.39953866600990295, -0.5782663226127625, 0.8751198053359985, 1.1895796060562134, 0.25094082951545715, -0.29824477434158325, -0.27331864833831787, -0.9008772373199463, -0.37397095561027527, 0.7788100838661194, 0.2738892436027527, 1.5203979015350342, 0.837818443775177, 0.4771052598953247, 0.1348266303539276, -0.3777446448802948, 0.42011767625808716, -0.40596839785575867, -0.05099991336464882, 0.4214313328266144, -1.6964062452316284, -1.2934027910232544, -1.3883311748504639, 0.5904127359390259, -1.347999095916748, 1.3549246788024902, 1.101426124572754], [1.900949239730835, 0.4467756450176239, 0.36037227511405945, 0.11561112105846405, -1.019812822341919, 0.7381757497787476, -0.5535725951194763, 0.27493077516555786, 0.6443209648132324, -0.5226515531539917, -0.989648699760437, 1.673044204711914, -0.7408210635185242, 1.4702634811401367, 0.40797293186187744, 0.6001653671264648, 0.08079476654529572, -1.1235263347625732, -0.6891221404075623, 0.3205549716949463, 1.3580986261367798, 1.0889755487442017, -0.8476076126098633, 1.5546702146530151, -1.380590558052063, -0.17261190712451935, -1.0776971578598022, -1.0811183452606201, -1.7710039615631104, -0.7575421333312988, -0.3458918631076813, -0.7824799418449402, -0.14077462255954742, 0.13361915946006775, 0.030722688883543015, -1.1959055662155151, 0.20275160670280457, -0.990625262260437, 1.5003634691238403, 0.45630598068237305, -0.16474105417728424, -1.6199570894241333, 0.4051029086112976, 0.44652989506721497, 0.009948430582880974, -0.005278251599520445, -2.1453146934509277, -1.7452847957611084, 2.8051085472106934, -0.5279327630996704, -1.3246716260910034, -0.3709582984447479, -1.079131841659546, 0.9151803255081177, -1.7468929290771484, 0.17544184625148773, -1.5602881908416748, -0.13774116337299347, 0.28134238719940186, -0.15946321189403534, -0.6393066048622131, 0.5011732578277588, -2.3082358837127686, -2.0165388584136963], [-0.4207991361618042, 0.046292588114738464, -0.6555613279342651, 1.6743377447128296, 0.8437321186065674, -1.5262796878814697, -0.3979485034942627, -0.181341752409935, -0.3209448754787445, 0.8184181451797485, 1.1739394664764404, 0.2279331386089325, -0.7097065448760986, -1.6525022983551025, 0.5209211111068726, 1.84321928024292, 1.4949684143066406, -1.8337286710739136, 1.3765145540237427, 0.8654876351356506, -0.03268660232424736, 0.9337916970252991, -1.0243942737579346, 0.5708262324333191, 0.6010255813598633, -0.8717961311340332, 0.03232912719249725, 1.1317572593688965, 0.7722612619400024, -1.880631446838379, 1.1673580408096313, 0.4615645706653595, 0.3367421329021454, -0.8834140300750732, 0.35967379808425903, -0.3013893961906433, 0.12107524275779724, 2.1521928310394287, 0.45853158831596375, 0.035953089594841, 1.8076342344284058, 0.32689139246940613, 0.793066680431366, 1.1273919343948364, 1.0142322778701782, 0.8083329200744629, 0.844648540019989, -0.40126192569732666, 1.5105445384979248, -0.8978200554847717, -0.9754933714866638, 0.936059296131134, 0.7690831422805786, 1.6958813667297363, -0.7182594537734985, 0.4689829349517822, -0.49684613943099976, 0.6380414962768555, -0.9137910604476929, 0.4610895812511444, 1.1105399131774902, -2.2930550575256348, -1.2805070877075195, 0.6040250062942505], [0.2674522399902344, 1.3181040287017822, 0.7065764665603638, -0.8862035870552063, 1.7104952335357666, 0.328929603099823, 0.4767017364501953, 0.2037263959646225, 0.4363257884979248, -0.1985696703195572, -0.43681806325912476, 0.5627553462982178, 0.8736092448234558, 0.6357761025428772, -0.9036141037940979, -0.7774817943572998, -0.6250661611557007, 0.6693276762962341, -1.2244415283203125, -0.4704657196998596, 0.0961749479174614, 0.6778995394706726, -0.8062835335731506, 1.1805788278579712, -1.1683688163757324, 0.9567234516143799, -0.9254685044288635, -0.43238914012908936, -0.2669198513031006, -2.6051738262176514, -0.4065338373184204, 0.009837922640144825, -1.094972848892212, 0.5848801732063293, -1.4552977085113525, 0.8258070349693298, -1.0797969102859497, -1.9003746509552002, 1.6374218463897705, -0.33947449922561646, 0.12534695863723755, -0.3829130530357361, 0.4936720132827759, -0.38003697991371155, 0.9680418968200684, -1.9752312898635864, 0.9332707524299622, -0.5302348732948303, 0.1805826723575592, -0.5046588778495789, 0.2672547698020935, -1.1184911727905273, 0.4013545513153076, -1.233628749847412, -1.1499073505401611, -0.8434690237045288, -0.21759222447872162, -0.9449096322059631, 0.36877942085266113, 0.09268175065517426, 0.28116098046302795, -0.41718584299087524, -0.29820725321769714, -0.6649888157844543], [1.367228388786316, -0.32145798206329346, -0.1485965996980667, 0.29088443517684937, -0.5967772006988525, 0.43256592750549316, -0.6534051895141602, -0.29792529344558716, 0.07184164971113205, -2.1030375957489014, 0.25996801257133484, -1.0865671634674072, 0.4680228531360626, -0.27436909079551697, -0.23076404631137848, -0.48164159059524536, -0.5878227949142456, -0.41432130336761475, -0.3757929801940918, 0.8609466552734375, 1.032089114189148, 0.5831798315048218, -0.5715627670288086, -2.575194835662842, -0.09698863327503204, 0.6304785013198853, 0.028551243245601654, -1.1735384464263916, 1.5813417434692383, -0.8073574304580688, 0.7954695820808411, -0.29517531394958496, 0.7840230464935303, 0.820248544216156, 0.15791326761245728, 1.28086256980896, -0.02053293213248253, 0.8015312552452087, 0.3809499740600586, 1.9794337749481201, -1.6154251098632812, -0.21607111394405365, -1.5758352279663086, 0.017184678465127945, -0.557842493057251, 0.3975927531719208, 1.0162534713745117, 1.9065532684326172, 0.2706008553504944, 1.0612177848815918, -1.1189595460891724, -0.997248649597168, 0.25697213411331177, -0.5942776203155518, 0.6556265354156494, 1.233052372932434, -0.48465341329574585, 2.705817699432373, 1.1899443864822388, -1.4099361896514893, 0.1029076799750328, 1.8763223886489868, -1.7027275562286377, 0.07674727588891983], [1.205742359161377, -0.8389531373977661, -1.6250324249267578, -0.7091248035430908, -1.0475114583969116, 0.6991472840309143, 0.9234454035758972, 0.9526605606079102, 0.48626261949539185, -0.0279061421751976, 1.1247622966766357, 0.013737645000219345, -0.2657887041568756, -0.8706809282302856, -0.6568593382835388, -1.1102676391601562, -1.276597499847412, -1.68344247341156, 1.7102762460708618, -0.335978239774704, -0.13422591984272003, 0.17027132213115692, -1.7111413478851318, 1.3624075651168823, 0.37590327858924866, 0.3306398391723633, 0.33066266775131226, -0.08871114999055862, -0.5941527485847473, 0.5521597266197205, -1.317531943321228, 0.5740928649902344, -1.1706911325454712, -0.3741411566734314, -0.7385395765304565, 1.803587555885315, -0.15737669169902802, -0.533650815486908, 2.220616579055786, 0.1402309536933899, -0.7399750351905823, 0.742225706577301, -0.13929834961891174, 1.3383220434188843, 2.2771520614624023, -1.823323369026184, -2.3928494453430176, 1.3059502840042114, 1.7685030698776245, 0.04381243512034416, -0.24350427091121674, 0.9368106126785278, -1.267769694328308, 0.03492511808872223, 2.0065369606018066, 0.3634595274925232, -0.3230993151664734, 0.7400853037834167, 1.112033724784851, -1.7335543632507324, 0.3560730218887329, -0.728398859500885, 2.221792459487915, 0.7665778398513794], [-0.5813540816307068, 0.8556003570556641, -0.37412434816360474, 0.2900567650794983, 0.25768712162971497, -1.093084454536438, 1.2301884889602661, 1.3943167924880981, 0.5907400846481323, -0.8615575432777405, -0.5251214504241943, 0.35661232471466064, -0.587699294090271, 0.3198261857032776, -0.49275121092796326, -0.8009601831436157, 0.21954445540905, -1.4775749444961548, -0.20478381216526031, -1.3112471103668213, -0.9143857359886169, 1.7293552160263062, -1.2737106084823608, -0.05603896453976631, 0.6396331191062927, -1.3525654077529907, 0.20344223082065582, 1.596853256225586, -2.5093750953674316, -1.4772919416427612, 0.19634531438350677, 1.2265530824661255, -1.7469251155853271, 0.7912452816963196, 0.2623024582862854, -0.5516957640647888, -0.7468230724334717, 0.49372386932373047, -0.09906625002622604, 1.21657133102417, -1.2672532796859741, 0.8755467534065247, -0.2499057799577713, -0.5141701698303223, -0.3823823034763336, -0.7191808819770813, 0.9285520315170288, 1.0927037000656128, 1.1317524909973145, 0.4561663568019867, 0.8558362126350403, -0.5821874737739563, -0.29748809337615967, -1.1263362169265747, -0.00876057893037796, -0.15379689633846283, -0.2184562385082245, 1.1103500127792358, -1.5280274152755737, 1.2982597351074219, 0.4793259799480438, -0.24453715980052948, -2.099590301513672, -0.48218628764152527], [0.3615063726902008, 1.319374442100525, 1.629636526107788, 0.5718615651130676, -1.095950961112976, 1.3909826278686523, -1.0170130729675293, -0.5548818707466125, -1.3915799856185913, 0.026609040796756744, -2.0205934047698975, -0.4827638864517212, 1.0130314826965332, -1.1253063678741455, -1.867881417274475, 1.5438437461853027, 0.9773507714271545, 0.9192264676094055, -0.9952208995819092, -0.3508071005344391, -1.1605606079101562, 1.401879906654358, -0.32027947902679443, -0.20131437480449677, 0.940459132194519, 0.9110580086708069, 1.8884271383285522, -0.859399676322937, -0.9450029730796814, -0.17344364523887634, -0.5194383263587952, 1.6641647815704346, 0.24244776368141174, 0.10210753232240677, 0.33510151505470276, -0.29411864280700684, -0.057845309376716614, -0.9695127606391907, -1.7394829988479614, -0.41341298818588257, 0.11754152178764343, 0.9347000122070312, 1.2132742404937744, 0.16346041858196259, -0.6618872880935669, -0.8527492880821228, 2.0893499851226807, -0.23022806644439697, -0.8737010359764099, 0.39606696367263794, -0.5424649119377136, 0.39607375860214233, 0.006976580247282982, -0.7916362881660461, 0.4394831359386444, -1.9368888139724731, -1.1796287298202515, 0.3442596197128296, 0.43615758419036865, -1.4545557498931885, -0.00812514778226614, 0.22784727811813354, -0.7839901447296143, -1.3222664594650269], [-0.5163864493370056, -1.7198559045791626, -1.2276134490966797, -0.8395814299583435, 0.6316876411437988, 1.1624754667282104, 0.6189969182014465, -0.22917622327804565, -2.3519399166107178, 0.772250771522522, 1.0023912191390991, -0.08580664545297623, 0.8387866616249084, 1.059593915939331, 0.6722732186317444, -0.6844775676727295, 0.2229483276605606, -0.49507710337638855, -0.7448529601097107, 1.809120774269104, 0.2673472464084625, -0.12720732390880585, 0.3670137822628021, -1.0075362920761108, 0.5944896936416626, -1.0886975526809692, 0.2926606237888336, -1.8109192848205566, -0.8038008809089661, 0.947640061378479, 0.6956981420516968, 0.17922402918338776, -0.6194892525672913, 0.45142662525177, -0.9568815231323242, -0.5616421699523926, -0.38536006212234497, 2.2303197383880615, -0.8392172455787659, -0.6528916358947754, 0.43222808837890625, -0.35970979928970337, 0.920395016670227, 0.027130044996738434, -0.23541902005672455, -0.7132415175437927, 1.064493179321289, -0.177149698138237, 0.681501567363739, 1.3672738075256348, -0.11137436330318451, 0.7653546333312988, 0.7463138103485107, -1.1877282857894897, -0.45842981338500977, 1.0278059244155884, -0.1183711588382721, 0.9323317408561707, 0.3467068076133728, -0.2449776828289032, -1.3027684688568115, 1.1250500679016113, -0.6525425314903259, 0.17675800621509552], [-0.028545834124088287, -1.0021384954452515, 0.779646098613739, -0.4075639545917511, 0.0898808166384697, 0.5213383436203003, -0.4929245114326477, 2.354088068008423, 0.3126608729362488, -1.0342035293579102, 0.6909342408180237, 1.546640396118164, 1.5469450950622559, 0.560940146446228, 1.187157154083252, -0.019243214279413223, 1.462208867073059, -1.9469640254974365, -0.48605427145957947, 1.3790315389633179, -1.1076927185058594, 0.2720957100391388, 0.8190709948539734, -0.18475879728794098, -0.4493139386177063, 2.246593713760376, 0.06792914867401123, 0.40277037024497986, -0.0757015272974968, 0.023156963288784027, 0.5646535158157349, -0.7593050003051758, -0.2790595293045044, 0.1575225293636322, 0.3980909585952759, 2.3373959064483643, -0.1274271458387375, -0.7955198287963867, 0.8144894242286682, -0.4862678647041321, -0.72805255651474, 1.0149712562561035, 0.8375518321990967, 0.16100367903709412, -0.4023926556110382, -0.49206289649009705, -1.953696608543396, 0.5169378519058228, 0.22989650070667267, -1.3949042558670044, -0.3822154998779297, 0.2658398747444153, -1.930738925933838, 0.9728697538375854, 0.09486430138349533, -0.4632323682308197, -1.3971896171569824, 1.177372694015503, 0.16300265491008759, 0.19376815855503082, 0.4190770983695984, -0.8341100215911865, 0.9516651630401611, 0.20133395493030548], [0.12617450952529907, 0.2894214689731598, 0.6812291741371155, 1.1984182596206665, 0.40403369069099426, -0.20346464216709137, -0.5426860451698303, 0.050816018134355545, 0.40174582600593567, -1.5104074478149414, 0.18538764119148254, -0.3488415777683258, -2.1611123085021973, -0.8529250621795654, 0.7918674349784851, 0.35184556245803833, 0.08561103790998459, 1.356890320777893, -0.8222104907035828, 0.21570591628551483, 1.3444981575012207, -0.44083157181739807, 0.9471685886383057, -1.4430345296859741, 1.9028180837631226, -0.5489885807037354, -1.1910295486450195, -0.2618371546268463, 0.6382437348365784, 0.21086522936820984, -1.5918177366256714, 0.015269310213625431, 0.1560373604297638, -0.36903613805770874, -1.4623935222625732, 0.16244296729564667, 1.6116176843643188, -0.8224416375160217, -0.13629120588302612, 1.3234049081802368, -0.11780904978513718, -0.4696512818336487, -0.3086817264556885, -1.6118488311767578, 0.2193245142698288, -0.5295675992965698, 0.3584183156490326, -0.22042185068130493, -0.18883182108402252, 0.7834581732749939, 1.1045161485671997, 1.1692183017730713, 1.4806623458862305, -0.6540306806564331, 0.6950296759605408, 0.6050403714179993, 0.40986424684524536, 1.7052040100097656, 0.979041576385498, -1.2096717357635498, -0.8840067386627197, -0.8085637092590332, 0.3206001818180084, 0.3775503635406494], [-0.20121733844280243, -0.20454223453998566, -0.7342126369476318, 0.1610717624425888, -1.5043476819992065, -1.1823339462280273, 1.151746392250061, 0.6695218682289124, 0.20634067058563232, 1.530618667602539, -0.9556047916412354, 0.7393745183944702, 0.6179133653640747, 0.30924370884895325, 0.1463707685470581, -0.5475610494613647, 1.4508659839630127, -0.7495353817939758, -2.6684205532073975, 0.511490523815155, 1.2087817192077637, 0.07996788620948792, -0.4439474046230316, -2.572441577911377, 0.688349723815918, -1.6247109174728394, 1.2357301712036133, -1.3789403438568115, -0.0752241313457489, -1.8906406164169312, 0.49611154198646545, -1.2338066101074219, -0.3764042258262634, 0.489388108253479, -0.8550238013267517, -1.0816705226898193, 2.336397171020508, -1.7529232501983643, -0.8270626664161682, 0.8759168982505798, -0.8465099930763245, 1.2082890272140503, -0.10176868736743927, -0.9412260055541992, 1.4464701414108276, -0.6240038275718689, -0.6196836233139038, 0.03738014027476311, 1.1211796998977661, 1.2353235483169556, 0.08334171772003174, -1.0427616834640503, 0.6722801923751831, -0.7758682370185852, 1.1767528057098389, -1.79392671585083, 0.7221540212631226, -0.036890171468257904, 1.559780240058899, -0.6612238883972168, 2.3115041255950928, 1.3920061588287354, -0.4493769407272339, -0.5240420699119568], [-1.4563778638839722, -0.5732652544975281, -1.0672428607940674, -1.341570258140564, -0.34146741032600403, 1.4804106950759888, -1.1071938276290894, 1.4122552871704102, 0.8744709491729736, -0.16651196777820587, 0.27632689476013184, -0.4321872889995575, 0.6625838875770569, -0.6417117118835449, -0.6447011232376099, 0.02663678675889969, -1.6787960529327393, -0.15104202926158905, -1.9822810888290405, -0.9762176871299744, 0.25825637578964233, 0.046370215713977814, 0.7531305551528931, 0.56308913230896, 0.30733522772789, 0.7316069602966309, 1.09260892868042, 1.487111210823059, 1.458988070487976, -0.5845959186553955, -0.5470805764198303, 0.03592273220419884, 0.8812475800514221, -0.369780033826828, 0.25701114535331726, -0.49624305963516235, 0.9033841490745544, -1.322714924812317, 0.34079742431640625, 1.2007230520248413, -0.9536341428756714, 0.6994239091873169, -0.6091129183769226, -1.1970940828323364, -1.2077699899673462, 1.0838505029678345, -1.2982021570205688, 0.6699514389038086, -0.6541107296943665, -2.2218337059020996, 0.18667908012866974, -2.046056032180786, 1.7125797271728516, 1.0759384632110596, 1.4087872505187988, -0.3800208866596222, 0.7090656757354736, -0.24476829171180725, 0.9719302654266357, 0.06198519095778465, 1.1801918745040894, 0.005008557345718145, -0.06211891025304794, -0.14060045778751373], [-0.463360071182251, -0.20967553555965424, 2.514251232147217, -0.33139073848724365, 0.3498688042163849, -2.025984525680542, -0.6868563890457153, -0.18393008410930634, -1.0791212320327759, 0.8670571446418762, 0.13603228330612183, 0.5418961048126221, -2.463324546813965, 0.0717443898320198, -1.0097612142562866, 1.6941287517547607, -0.8116483092308044, 0.3097677528858185, -1.4524513483047485, -0.7477652430534363, 0.4509604871273041, 0.47830337285995483, 1.4397408962249756, 0.029653305187821388, -1.5201810598373413, 0.5325422883033752, -0.4893263876438141, 0.18116644024848938, -1.635312795639038, -0.28063395619392395, 1.599420189857483, 0.7797812819480896, -0.11510185897350311, 0.25011712312698364, -2.1079113483428955, -0.0694250538945198, -2.262805938720703, 0.046175021678209305, -1.5066872835159302, 0.03242422640323639, 0.6799699068069458, -1.5792771577835083, -0.7653785347938538, -0.10492806136608124, 1.772879719734192, 0.30280372500419617, 0.19423438608646393, 0.6917457580566406, -1.1168701648712158, -0.14962005615234375, 0.3736498951911926, -1.2281718254089355, -0.6464177370071411, 0.8742886781692505, 0.1984650045633316, -1.9425580501556396, -1.1276549100875854, -0.27208641171455383, 0.8000991940498352, -0.4234555959701538, -2.449259042739868, -0.20537248253822327, 1.8940848112106323, -1.331640601158142], [0.1397516429424286, 0.8015386462211609, 0.2687387466430664, 1.6487014293670654, -0.10342120379209518, -0.13308100402355194, 0.29319894313812256, 0.8227534890174866, 0.8115645051002502, -1.2957563400268555, -0.24178609251976013, 0.698742687702179, -1.549063801765442, 0.7537407875061035, 0.5539694428443909, 0.6432309150695801, -0.7896713018417358, 0.5012023448944092, 0.6599085330963135, 0.7172912359237671, 1.6231647729873657, -0.8263213634490967, 1.002963900566101, -1.29800283908844, -0.2511361539363861, 1.0819592475891113, -1.412696361541748, 0.5550794005393982, -1.1255944967269897, -0.4495263397693634, -1.4565138816833496, 2.0757904052734375, -0.5255088806152344, 0.013385646045207977, 0.08860044926404953, -0.6890131235122681, 0.2243543565273285, -1.0756057500839233, 0.6550514698028564, -0.07044573128223419, 0.3670165538787842, 1.2667491436004639, 2.4152767658233643, 0.7568383812904358, -0.7995436787605286, 0.1458001434803009, 0.14126750826835632, 0.9430923461914062, -0.5699856877326965, -0.7847609519958496, -0.5510757565498352, -0.8593519330024719, -0.43214547634124756, -1.1539076566696167, 1.5799775123596191, 1.5062296390533447, -0.5790448188781738, 2.013200044631958, -0.31895825266838074, -0.3790299594402313, 1.0081897974014282, 0.7954311370849609, -0.027084220200777054, 1.8068468570709229], [-0.6170351505279541, 1.4707653522491455, -0.6529309153556824, -0.8341588973999023, -1.0136220455169678, 0.5497410893440247, 1.101292371749878, 2.166175127029419, -0.9934470057487488, -1.0136134624481201, -0.41729071736335754, 0.7484056949615479, -1.0282881259918213, 0.05270334705710411, 1.1406831741333008, 2.479631185531616, 0.6507920026779175, -0.31104570627212524, 0.08761933445930481, 0.3343266248703003, -1.324516773223877, -0.33928030729293823, 0.04552914574742317, -0.877525269985199, -1.1078248023986816, -0.5066205859184265, -0.070210300385952, 1.1147968769073486, -0.9969066381454468, -0.09968239068984985, 1.3574374914169312, -0.32898449897766113, 0.42794540524482727, 0.09338483214378357, -1.0320079326629639, 0.3904019892215729, 0.2967359721660614, -0.8826578259468079, 1.5666985511779785, -0.16610108315944672, 0.9419623017311096, -2.0286669731140137, 0.2546026110649109, -1.2810156345367432, 0.5977205634117126, -0.32203057408332825, -2.0383105278015137, 0.30707961320877075, -1.256295084953308, 0.8486009240150452, -0.6069908738136292, 0.9478394985198975, 0.6755830645561218, 0.5009061694145203, 0.48529714345932007, -2.841557741165161, 0.16527409851551056, 1.141494870185852, 0.08303005993366241, -0.6459311842918396, -1.19118070602417, 1.6753127574920654, -1.0881975889205933, -1.622205138206482], [-0.8009845018386841, -1.3315331935882568, 0.6733044981956482, -0.20573441684246063, -0.01697503961622715, -1.7614974975585938, 1.3819619417190552, 0.987342894077301, -1.6744482517242432, 0.1407133787870407, 0.5807012915611267, 2.7073686122894287, -0.8823391199111938, -1.018277883529663, 0.10936100780963898, -0.7371132969856262, -0.39777520298957825, -0.23926348984241486, 0.40392377972602844, 0.10895302146673203, -0.687449038028717, 0.8722869157791138, 0.08245395869016647, -0.05385132506489754, -0.4830230474472046, 0.22821448743343353, -1.5805867910385132, 0.2022913247346878, 0.8385154604911804, 0.4858040511608124, -0.7904430031776428, 0.10732849687337875, -1.4274100065231323, 1.1492292881011963, -0.9204935431480408, -1.0917999744415283, -1.3922150135040283, -1.2034032344818115, 0.6779810786247253, -1.996727705001831, 0.23103679716587067, -0.05117645114660263, 2.0401265621185303, -0.5699070692062378, 0.2577950060367584, 0.34561094641685486, 0.37362250685691833, -0.919806718826294, 0.24016301333904266, -0.07146721333265305, -0.0983256921172142, 0.16838271915912628, -0.08999275416135788, 0.9125741720199585, -0.3131030201911926, 0.14151346683502197, -0.687055766582489, 2.1115000247955322, -0.7463217377662659, -1.3591481447219849, 0.173211932182312, -1.197344183921814, 1.9268945455551147, -1.6324870586395264], [0.2247227132320404, 1.257867693901062, 0.10313485562801361, 0.26611700654029846, 0.5390051603317261, 0.7718654870986938, -2.1164326667785645, -0.9536339640617371, -1.0723515748977661, -0.20618820190429688, 0.5210697054862976, -0.13611187040805817, 2.1884472370147705, 0.21900318562984467, -0.6300839185714722, 1.9574626684188843, -0.7654738426208496, -1.3873602151870728, 0.3986947238445282, -0.20188447833061218, -0.42571383714675903, -1.9997508525848389, -0.39815181493759155, -0.22422939538955688, -0.0092232720926404, 1.1179039478302002, 0.34951040148735046, 0.2004098892211914, 0.3740466237068176, -1.8446422815322876, 1.640375018119812, -0.8194252252578735, -0.0582343190908432, -0.1992291361093521, 2.2285149097442627, -1.5023233890533447, -0.9986978769302368, 0.5101383328437805, -2.2704317569732666, 0.14777761697769165, -0.9674867391586304, -0.01500025112181902, 1.2618553638458252, -0.9334543347358704, 0.9755429625511169, 1.129642367362976, 0.24308352172374725, -0.3976759612560272, 0.12468058615922928, 0.5178859829902649, -1.670961618423462, 0.4914362132549286, -0.7620752453804016, -0.44613125920295715, -0.9738385081291199, 1.0425245761871338, 0.4287787973880768, -1.637630820274353, -0.07263021171092987, 0.6525689363479614, 0.10018706321716309, -3.3486483097076416, -0.32977229356765747, -1.3852883577346802], [2.617396116256714, 0.5512712597846985, 1.0245940685272217, -1.2222756147384644, 1.4075907468795776, -1.3823198080062866, 0.7682731747627258, -1.2713630199432373, 0.7291779518127441, -0.5286149978637695, 0.24651095271110535, 1.1761858463287354, 0.35609200596809387, -0.440412312746048, 0.9223191738128662, 1.124733567237854, 0.11506529897451401, -0.04591327905654907, 1.8946915864944458, 1.0149706602096558, 2.609013080596924, 1.8303136825561523, 0.2265390157699585, 0.41483691334724426, 1.7988866567611694, -1.5507224798202515, -0.5409899353981018, 0.5004736185073853, 1.0748867988586426, 0.23795944452285767, 0.19992756843566895, 0.2815362215042114, -1.45041024684906, -0.8313435912132263, 0.6886240243911743, -0.6077802777290344, -0.08700534701347351, -1.0901689529418945, 0.693004310131073, -0.33434584736824036, -0.1404670923948288, 0.4185074269771576, 0.8777519464492798, 1.1265040636062622, 1.1886075735092163, 0.6612245440483093, -0.5995994210243225, 0.7219438552856445, 1.7926453351974487, 1.3554553985595703, 1.0103018283843994, 2.165510892868042, 0.5724643468856812, 3.114445924758911, -0.7226824164390564, -1.0859519243240356, -0.22191961109638214, -1.8143235445022583, -0.7863307595252991, 0.5345485806465149, 1.144694447517395, -0.9327226281166077, 0.7907547950744629, 0.48803555965423584], [-0.17103168368339539, 0.2543430030345917, -1.3829199075698853, 1.2220840454101562, -0.24930915236473083, -0.045673783868551254, -1.2918659448623657, -1.740829586982727, -0.06902298331260681, -1.4605066776275635, 0.26651668548583984, 1.1394542455673218, 1.1958484649658203, 1.724579930305481, 0.171957328915596, 1.306768536567688, -0.8702545166015625, 0.8611926436424255, -0.08467042446136475, -0.07671958208084106, 0.47009366750717163, 1.1479555368423462, -0.7339476346969604, 0.9364402890205383, -1.3414509296417236, 1.6818917989730835, -0.016591563820838928, -0.801114559173584, -0.7750391364097595, -0.36473503708839417, -0.27907851338386536, -1.9585403203964233, 0.07419376820325851, 0.7155402898788452, -1.9606214761734009, -0.207102432847023, -0.04209037125110626, 0.7805504202842712, -1.1405670642852783, 0.5508759021759033, 1.6713401079177856, -0.6742705702781677, -2.284555196762085, -1.1232266426086426, -0.23657941818237305, 0.46517154574394226, 0.5842460989952087, 0.4795829653739929, -1.1907002925872803, -0.7384862899780273, 0.040666352957487106, -0.33273348212242126, -0.7635565400123596, 0.5674821138381958, 1.167236089706421, 0.004595273174345493, 0.42211541533470154, 0.062308378517627716, -0.5736062526702881, 0.35527774691581726, -1.0243470668792725, 1.7625741958618164, 0.5466451644897461, -0.3313900828361511], [-0.5734077095985413, -1.6823463439941406, 0.18746285140514374, 1.1129350662231445, -1.5387725830078125, -0.6614409685134888, -0.24422307312488556, -0.34098175168037415, 0.1976310759782791, -1.463809609413147, 0.4145355522632599, -0.5960413217544556, -1.283614993095398, -1.0001893043518066, -0.4563647210597992, 1.023010492324829, -0.045186031609773636, -1.1168917417526245, -1.133902907371521, 0.06146533787250519, 1.2612059116363525, 1.050769567489624, 0.5692328214645386, -0.47790664434432983, 1.3083326816558838, -1.708340048789978, 0.8146132826805115, -0.8045053482055664, -0.6437041163444519, 0.14393529295921326, -0.08571130782365799, -1.442742943763733, 0.6720370650291443, -1.1614848375320435, -0.7256619334220886, 0.19761161506175995, 0.6027634739875793, -0.6586812138557434, -0.002579964930191636, -0.07120338827371597, -0.6079975366592407, 0.14863483607769012, 0.3505830466747284, -1.1665843725204468, -0.24879957735538483, 0.5077890157699585, -0.03872663155198097, 0.3233346939086914, 0.3995198905467987, 0.7156216502189636, -1.2346522808074951, -0.568855881690979, -1.0826324224472046, -0.6128735542297363, -1.1986833810806274, -0.44695475697517395, 0.0005633348482660949, 0.6480427384376526, 0.017009984701871872, 1.157270908355713, 0.28483086824417114, 0.09775418043136597, 0.24673718214035034, 0.7519261240959167], [1.2212978601455688, 0.2415003925561905, 1.453302025794983, 0.5890039205551147, 0.5512621402740479, -0.546298086643219, -0.48939186334609985, -0.9779444932937622, 0.7158851623535156, 0.6385387182235718, 1.8531776666641235, -1.1739469766616821, -0.7326381802558899, 0.11543504893779755, 1.2037442922592163, 0.7752962112426758, -0.6225302815437317, 0.331500381231308, -0.3958651125431061, 0.6775487065315247, 0.13495109975337982, -0.20139949023723602, -0.8094442486763, -3.1091127395629883, 0.40379565954208374, 1.1378480195999146, -1.809748649597168, -0.7164259552955627, 1.2368042469024658, -0.5459105372428894, -0.85923832654953, 0.5967039465904236, 1.0509355068206787, 0.5172690153121948, 0.22125238180160522, 0.7455171346664429, 0.8525059223175049, 0.5862293839454651, 0.4205048978328705, -0.8346623778343201, -0.5172337293624878, -0.3340272307395935, 0.7347251176834106, -0.916650116443634, -0.07497947663068771, -0.8499494791030884, 0.13128982484340668, -0.031708188354969025, 0.4342498183250427, -1.2044953107833862, 0.4911494851112366, 0.6449965238571167, -1.296303391456604, 1.800833821296692, 1.8218361139297485, 0.5184480547904968, -1.4659334421157837, 0.1938585638999939, 0.6065451502799988, -1.611544132232666, 2.2008557319641113, -1.3529846668243408, 0.5445378422737122, -2.469569444656372], [-0.2021830826997757, -0.3067655861377716, -0.18061204254627228, -0.879330575466156, 0.33741623163223267, 1.0348583459854126, 0.5652144551277161, -0.5110487341880798, -0.05687231197953224, -0.6645749807357788, 0.17087861895561218, 0.5147292613983154, 0.4201852083206177, -1.1554317474365234, 0.02601265162229538, 0.7152373194694519, -0.5999053716659546, -0.5050066709518433, -1.182002305984497, -2.240623712539673, -0.908661425113678, 0.7313355803489685, -0.8267528414726257, -0.27391064167022705, 0.6271917819976807, -0.4295908510684967, 1.1137288808822632, 0.2623153030872345, -0.8465993404388428, 0.22683623433113098, 1.2785054445266724, 0.7177979946136475, 1.3912866115570068, -2.982172727584839, -1.0921622514724731, -0.5427355766296387, 0.04000517353415489, -1.7746691703796387, -0.46837127208709717, -1.3252522945404053, -0.5242611169815063, 1.4411535263061523, 0.4660777151584625, 0.2649537920951843, 0.6726779341697693, -0.7617570161819458, -0.11720098555088043, -0.7478209137916565, -0.7131417989730835, -0.4585503339767456, 0.7062056064605713, 0.45976734161376953, 1.5651209354400635, -0.3479124903678894, -1.266859769821167, -1.1998218297958374, -2.073848009109497, 0.4899158477783203, 0.5720356106758118, 0.682634174823761, 0.7974684238433838, -0.1507517695426941, 0.6357163786888123, -0.5571491718292236], [-1.1659290790557861, 0.42756202816963196, -2.1636388301849365, 1.382707953453064, 0.7200840711593628, 0.5237117409706116, 0.37719398736953735, 1.323968768119812, 0.5470861792564392, -0.9770590662956238, -1.1514660120010376, 0.16134344041347504, -0.31652578711509705, -0.9652442336082458, -0.006161890458315611, 0.5227288603782654, 2.5624923706054688, -0.8320498466491699, -0.7105538845062256, 0.09070821851491928, -0.04158885404467583, 1.215267300605774, -0.10968556255102158, 0.16040372848510742, 1.101704716682434, -1.2728325128555298, 0.3150014579296112, 0.6737840175628662, 0.4697848856449127, 1.0464569330215454, 0.5066291689872742, -1.3118665218353271, 0.3867085576057434, 1.5956922769546509, -1.8718557357788086, -1.1194908618927002, -0.46212315559387207, 1.1278084516525269, 0.0312853641808033, -0.5302723050117493, -0.39369311928749084, 0.749032735824585, 0.05174590274691582, 1.0702590942382812, 1.4663968086242676, 1.5738182067871094, 0.4811818301677704, -0.7856801748275757, 0.23839779198169708, -0.6016449332237244, -0.17827332019805908, 0.44227322936058044, 0.7651306390762329, -1.126228928565979, 0.7760009169578552, -0.22644521296024323, 0.7836200594902039, -0.4385996162891388, 0.8642442226409912, -1.508360743522644, -1.6921050548553467, 0.792205810546875, 1.8554741144180298, -0.4860125184059143], [-0.22276844084262848, -1.7845350503921509, -0.18186376988887787, 0.04269788786768913, -1.6526436805725098, 1.02168869972229, 0.8757303953170776, -0.296794056892395, -0.3825884461402893, 0.45383337140083313, 0.708964467048645, 0.5397118330001831, -1.4001954793930054, -0.0859614908695221, 0.278204083442688, -1.277682900428772, 0.14901039004325867, 0.25489622354507446, 0.6035386323928833, -0.010061013512313366, -0.10972440242767334, -0.38771551847457886, -0.05072196573019028, 2.114863157272339, -0.4425778388977051, -0.8753530979156494, -0.6189227104187012, 0.036820054054260254, 0.5120838284492493, 0.22446446120738983, 1.1541301012039185, 1.3206124305725098, 0.19951438903808594, -0.19733040034770966, -0.16691765189170837, 0.4432998299598694, -1.0012842416763306, 0.4286547899246216, -0.13852407038211823, 0.36982518434524536, 0.1753583699464798, -0.23461218178272247, -0.2643292248249054, -0.7951655983924866, -0.5530161261558533, 1.3760277032852173, 0.18721963465213776, -1.0004228353500366, 0.1116376742720604, 1.298967957496643, -0.6878083348274231, 0.5273145437240601, -0.30600428581237793, -0.6247766017913818, -0.9755973815917969, 0.28834736347198486, 1.2874183654785156, 0.5684224963188171, 0.03730172663927078, -1.1963040828704834, -1.4611696004867554, 0.5360748767852783, -0.5375065207481384, -0.05994601175189018], [-0.7694064974784851, 0.13021087646484375, -0.7363366484642029, -1.5838160514831543, 1.5729146003723145, -0.42123907804489136, -0.16940096020698547, -0.717496931552887, -1.7131670713424683, 0.1061193123459816, 0.03353085741400719, 0.8009239435195923, -1.8332948684692383, -0.7261531352996826, -0.514327347278595, 0.11282126605510712, 0.3847867250442505, -0.6465979218482971, -0.6037167906761169, 0.26966726779937744, -1.3756129741668701, -0.23001162707805634, -2.087128162384033, 0.656582772731781, 1.9060486555099487, -0.40321576595306396, 1.2795331478118896, 0.45238733291625977, 0.5842148065567017, 0.45030173659324646, 1.5060889720916748, -0.4525139033794403, 0.5158730745315552, -0.3239470422267914, -0.5226916670799255, 0.909542977809906, -1.3380649089813232, 1.6023592948913574, 0.07470838725566864, 0.5770578384399414, -1.5302965641021729, 0.611666738986969, -0.9928207397460938, -1.8846365213394165, -1.0613999366760254, -1.2704392671585083, 1.2763645648956299, -0.8892643451690674, 0.36573082208633423, -0.8032501339912415, 0.030051883310079575, 0.18665842711925507, -0.4625983238220215, -0.5553697943687439, 0.28691384196281433, 0.5017483830451965, 1.0273321866989136, 2.866206169128418, 0.8603756427764893, -1.258162498474121, -0.27794399857521057, 0.18450455367565155, 0.7456607818603516, -1.9705911874771118], [0.7988564372062683, 1.193328857421875, 0.19690315425395966, 0.0740252211689949, -0.5361290574073792, -0.47524556517601013, -0.4319307506084442, 0.9935755133628845, -0.08708982914686203, -0.08176420629024506, 0.9868924021720886, -0.5220907926559448, 0.556734561920166, -1.9109430313110352, -0.48336416482925415, -0.7261762619018555, -0.4178940951824188, -1.9647899866104126, -1.2713602781295776, 0.3979891240596771, 1.5452260971069336, -1.4060128927230835, 0.7293123006820679, -0.14555253088474274, -1.502690076828003, -1.524917483329773, 1.1814196109771729, -1.285443663597107, -0.7679304480552673, 0.8231741786003113, -1.062218189239502, -2.407050371170044, 1.0897388458251953, 0.366889625787735, 0.9316253066062927, 1.2761516571044922, -1.0206305980682373, -0.6797441840171814, -2.1318888664245605, -1.5094183683395386, -0.5596622228622437, 0.6153606176376343, -1.2305606603622437, 0.3096209168434143, 0.8984935879707336, -0.6038017272949219, -0.06994106620550156, -0.20078210532665253, -0.47862932085990906, 1.0535119771957397, -0.2114986777305603, 0.8339297771453857, 0.1915222853422165, 1.023088812828064, 1.3955190181732178, 2.533360481262207, -1.9306583404541016, 0.5890363454818726, 1.0294495820999146, -0.044851526618003845, 0.6412192583084106, 1.1693111658096313, -1.5175988674163818, -0.3725581467151642], [-1.239828109741211, -0.2869562804698944, 0.7773798108100891, 0.24243564903736115, -0.3075449466705322, 0.2784387171268463, 1.0108940601348877, -0.19140370190143585, 0.16900213062763214, 0.4072268605232239, 1.341480016708374, 0.9857786893844604, -1.2641764879226685, 0.7931694984436035, -0.7015683054924011, 0.7347723245620728, -1.8141013383865356, 0.292097806930542, 0.10081474483013153, 0.5972521305084229, -0.12471959739923477, -0.15518654882907867, 1.7969732284545898, -1.4021202325820923, 0.3797501027584076, 0.6012431383132935, 0.32828500866889954, -0.5378047823905945, -0.02664651907980442, -2.1087486743927, -0.20312483608722687, -1.5968379974365234, -2.689265012741089, -1.1002991199493408, -2.607041120529175, -0.8058302998542786, -0.8583022952079773, 0.4187883138656616, 0.4185980558395386, 0.2800937294960022, -0.8137078285217285, 0.7499759197235107, -1.1456187963485718, 0.7433709502220154, 1.0056302547454834, -0.9554649591445923, -0.1693294197320938, -0.540976881980896, -1.1124597787857056, 0.28208035230636597, 0.9256726503372192, -0.2819044291973114, 0.5221885442733765, -0.2783827483654022, -0.9912043213844299, 0.06670156121253967, -2.4906632900238037, -1.5011999607086182, 1.5522427558898926, 0.9267452955245972, 0.17515867948532104, -0.11936803162097931, 1.2275621891021729, -0.9632741808891296], [1.03883695602417, -1.0496236085891724, -0.5930289626121521, -2.164388418197632, 0.06051791086792946, 0.41029784083366394, -0.9166962504386902, 0.2711038589477539, -1.090724229812622, 0.47395211458206177, -0.944943904876709, 0.49388477206230164, -2.3222339153289795, -2.0396249294281006, 0.9728251099586487, 0.22192400693893433, -1.1914440393447876, 0.20123693346977234, 0.1923701912164688, 0.129769966006279, -0.18068768084049225, 0.992447555065155, -1.0909591913223267, 1.160261631011963, -1.0158239603042603, -0.1496221125125885, 0.8978543877601624, -2.5967743396759033, -1.0479681491851807, 2.660074234008789, -0.5139033794403076, 0.5775235295295715, -0.3707302212715149, -0.4198581874370575, 0.6236894130706787, 1.1337761878967285, -0.12718068063259125, 1.3107614517211914, 0.32523083686828613, 1.3131269216537476, -0.15805421769618988, -0.12082836031913757, -0.5461025238037109, -0.21217146515846252, 1.2680104970932007, 1.919726848602295, 0.8689979910850525, 1.5863032341003418, 1.028075933456421, -0.9935989379882812, 0.641723096370697, 1.2740312814712524, 1.4141634702682495, -0.19763702154159546, 1.4077168703079224, -0.11451637744903564, 0.4325586259365082, -1.590035319328308, 1.621715784072876, 0.4223586618900299, 2.9553864002227783, 0.8697800040245056, -1.7253015041351318, 0.6139346361160278], [-1.033380389213562, 1.4520002603530884, 0.007830280810594559, -0.7813035249710083, -0.16157092154026031, 0.8944870829582214, -0.22087910771369934, 0.9141169190406799, -1.407528281211853, -0.9432435631752014, 0.5364949703216553, -0.019626246765255928, -0.05680031329393387, -1.2413866519927979, -0.054199639707803726, -0.017283478751778603, 0.02532411552965641, 1.7592800855636597, 0.7601848244667053, -1.255184292793274, -0.0357295423746109, -0.251403272151947, -1.8426164388656616, -0.3925776183605194, 0.8630552291870117, -1.899564266204834, 0.5900076031684875, -0.36841726303100586, -0.10226795822381973, -0.4190760552883148, -0.7121186256408691, -0.5292938947677612, -0.442561537027359, -0.9876326322555542, 1.9522762298583984, -0.12515360116958618, -0.07599316537380219, 0.31825539469718933, 0.13010424375534058, -0.3935073912143707, -0.18075771629810333, 1.399606704711914, 0.6485069990158081, 0.7060132026672363, 0.3288191854953766, 0.22967636585235596, -0.08742865175008774, 1.197507619857788, -0.2060643434524536, 0.1152462288737297, 0.5091202259063721, -1.6430803537368774, -2.216073989868164, 0.03684258833527565, 0.7539827227592468, 1.099379301071167, 1.1348613500595093, -0.3236769139766693, 0.4678216874599457, 0.6079908013343811, -0.5396056771278381, -1.0340932607650757, 0.6879174113273621, 0.9181904792785645], [0.24092894792556763, 0.6960617303848267, 0.28689926862716675, -0.6485148668289185, 0.6210349202156067, -0.15184806287288666, 0.005188025068491697, 1.3490077257156372, 0.816897988319397, -0.3563104569911957, 0.1778959482908249, -0.9005025029182434, -1.799832820892334, 0.8233096599578857, 0.4043724536895752, 1.6007617712020874, 0.5796425938606262, 0.8509238362312317, -0.4088573753833771, -2.0392680168151855, 0.25340554118156433, 0.49874621629714966, -0.1506129652261734, 0.3979254961013794, -1.1141830682754517, -0.03973356634378433, -0.8985902070999146, -0.34729087352752686, 1.6428592205047607, -0.07302232086658478, 0.09665752947330475, 0.10206739604473114, -0.6533073782920837, 0.8980739116668701, 1.9149928092956543, 0.02244744822382927, 1.2212364673614502, 1.1534037590026855, -1.9571657180786133, -0.7026095986366272, -0.31717967987060547, -0.6351468563079834, -1.5948578119277954, 0.5488138198852539, 0.5987435579299927, -2.161801338195801, -0.6638704538345337, 1.1126306056976318, -0.46381229162216187, -1.2379637956619263, 0.24349121749401093, -0.11118802428245544, 0.8907507061958313, 0.6416997313499451, 0.10450772196054459, 1.3099831342697144, 1.2351433038711548, -0.5352671146392822, -0.5747154951095581, -1.208443284034729, 1.4522830247879028, -2.0605623722076416, 0.7279469966888428, 1.0494561195373535], [0.5228856205940247, 0.7804082632064819, 2.621997833251953, -1.1484732627868652, 0.9582280516624451, -2.0645320415496826, 0.0440838597714901, -1.2373244762420654, -0.6433919668197632, 1.449075698852539, 0.6142643690109253, 0.745384693145752, 0.6428982019424438, 1.490256667137146, -0.4832388162612915, 2.9640722274780273, -0.8208906650543213, 0.15412108600139618, -0.5027691125869751, -0.4217259883880615, -0.6388852000236511, 0.3795728087425232, 0.5879064202308655, 1.6187950372695923, -1.966746211051941, 0.38384148478507996, 0.635015070438385, 1.032508373260498, 0.14943180978298187, 0.07652100175619125, -1.4348872900009155, 0.1980903148651123, -0.3250615894794464, -0.27545011043548584, -0.9696400761604309, 1.5218855142593384, -0.6754167079925537, -0.6472035646438599, -0.7426499128341675, -1.3380300998687744, 0.3958165943622589, 0.2549205422401428, -1.3344874382019043, -0.6281467080116272, 1.0678447484970093, -0.954708993434906, 0.23160101473331451, 0.4801327884197235, 0.49168625473976135, 2.0210139751434326, -0.21371807157993317, -1.066603183746338, 2.2412214279174805, -0.8031569123268127, 1.3054455518722534, -1.221818208694458, 0.03895058482885361, -0.5115060806274414, -0.15229351818561554, -0.33813196420669556, 0.19628331065177917, -1.2762068510055542, 0.4438849687576294, 0.3405236005783081], [-1.467545986175537, -0.41446536779403687, -0.0707782432436943, 0.7282090187072754, -2.4942498207092285, -1.0710852146148682, 0.8726925849914551, 0.9714455604553223, 1.2108869552612305, 0.41656315326690674, -1.5030877590179443, 0.3126557767391205, 0.339207261800766, 0.2867773771286011, 1.1437602043151855, 1.1289464235305786, 0.29021090269088745, 0.45529505610466003, 0.29068681597709656, 0.54034823179245, -0.6568037867546082, -0.34146571159362793, -0.1763078272342682, -0.6194270849227905, 1.1143288612365723, 0.29515540599823, -0.7030161023139954, -0.8428546190261841, -0.6358534097671509, 1.38141667842865, -0.12993167340755463, -0.5518805384635925, -0.3090231716632843, -0.8822148442268372, 0.7957842946052551, -1.380223274230957, -0.5927712917327881, 1.0159885883331299, 1.277513027191162, 0.8250393867492676, 0.33323338627815247, 0.045153237879276276, 0.18283692002296448, -1.1988312005996704, 0.7550380229949951, -1.1055259704589844, 0.5884299278259277, -0.05093494430184364, 1.4444845914840698, -0.2604405879974365, 1.0901256799697876, 1.5381121635437012, 0.11651629209518433, 0.5633707046508789, -1.2198450565338135, 2.13535475730896, 1.5598161220550537, -1.849732756614685, -0.37346598505973816, 0.331439346075058, 2.1411118507385254, -0.7810350656509399, 1.1959327459335327, -0.2850932478904724], [2.1161513328552246, -0.46057042479515076, 1.2195638418197632, -0.7607629895210266, 0.062443945556879044, 2.0907018184661865, -0.6741066575050354, 2.0050246715545654, 0.8637510538101196, 0.8896892666816711, -0.017260031774640083, 1.4913012981414795, 0.8070964813232422, 0.5274898409843445, 1.5549737215042114, -1.0414477586746216, -2.4760000705718994, -0.18525899946689606, -0.5517205595970154, -1.7666780948638916, 0.25389960408210754, 0.710161566734314, 2.240037679672241, -1.4506969451904297, 1.6570318937301636, -0.5728054642677307, 0.21216799318790436, 1.1904886960983276, 0.4239075183868408, -0.004056867677718401, -0.3024761378765106, -2.054009437561035, 0.21144424378871918, 1.043365716934204, 0.6873080730438232, -2.116666078567505, -0.21091167628765106, 0.8317969441413879, -0.9694872498512268, -0.574166476726532, -1.2971566915512085, -0.3174694776535034, 1.3247994184494019, -0.1496298462152481, 0.6497030854225159, -1.2986024618148804, -0.7076087594032288, -1.7499781847000122, 1.8045257329940796, 0.6525537967681885, -1.2573293447494507, 0.6177513003349304, -0.27040910720825195, 0.3248341977596283, 1.1551733016967773, -0.2785210907459259, 0.682445764541626, 0.29552653431892395, -0.9366089701652527, 1.6360846757888794, 0.23658587038516998, -0.851723313331604, -0.9790770411491394, -0.1851452738046646], [0.8703217506408691, 0.6213094592094421, -0.15527652204036713, -0.27798324823379517, -0.1802452951669693, -1.0271903276443481, 0.42837169766426086, 1.423717975616455, 0.5445855259895325, -2.2310101985931396, 0.6226897835731506, -0.08624125272035599, -0.3526729643344879, 0.24102076888084412, 0.33566904067993164, -0.1817443072795868, 0.35161688923835754, -1.7136093378067017, -0.49719876050949097, 0.9645054936408997, 2.118257522583008, -0.12492283433675766, -0.417476087808609, 0.31872013211250305, 0.16524438560009003, 0.6680042743682861, 0.3116520345211029, 0.27609506249427795, 0.6182371377944946, -0.3151053488254547, 0.9680827856063843, -1.1585149765014648, -0.3083433508872986, 0.8737510442733765, -1.2021760940551758, 0.2304408699274063, -0.9360228180885315, 0.7803433537483215, -1.5696358680725098, 1.3523719310760498, 1.2213470935821533, 2.7157795429229736, -0.06800144165754318, 0.571063220500946, -2.155557632446289, -0.2851638197898865, 2.1486222743988037, -1.7132630348205566, -1.2219538688659668, -0.47966358065605164, -1.1968098878860474, 0.3036416471004486, -0.08828739076852798, 1.4161927700042725, 0.864750325679779, -0.9376161098480225, -0.41976311802864075, -0.1349441558122635, 0.136018306016922, 1.5897464752197266, 0.09861357510089874, 0.17766308784484863, 1.4608485698699951, 0.03788289800286293], [-0.781765878200531, -0.9078922271728516, 1.2301044464111328, -1.3980098962783813, -1.4511605501174927, -0.011703331023454666, -0.06221729889512062, 0.4126432538032532, 0.25620484352111816, 0.18898802995681763, -0.7982829213142395, -0.8448700904846191, -1.5815684795379639, -0.12836475670337677, -0.16303974390029907, 0.4533196985721588, -0.12909454107284546, -0.4667075276374817, 1.328734040260315, 0.10469608008861542, -0.9895725846290588, -0.975639820098877, -0.1651543229818344, -0.0384678840637207, -1.5330315828323364, 0.19906195998191833, 0.5841711759567261, -0.600892186164856, 0.013828463852405548, 0.6120505332946777, 0.9558265805244446, 1.3781049251556396, -0.04897664859890938, -0.06519574671983719, -1.4756985902786255, -2.3358278274536133, -0.4399130940437317, -0.3096689283847809, 1.3234553337097168, -0.5748513340950012, -1.5705097913742065, 0.6385457515716553, -0.10002526640892029, -0.9553554654121399, 0.37640225887298584, 0.3629450798034668, 0.07436446100473404, -0.5214921236038208, -1.2725837230682373, 0.19817811250686646, 0.9855635166168213, 0.5977102518081665, -0.563088595867157, 1.2271398305892944, 1.1247342824935913, 2.409900188446045, 0.4855063259601593, -1.015397548675537, -0.0742775946855545, -0.1250530332326889, 0.7189216613769531, -0.5932019352912903, -0.08512231707572937, -0.15854685008525848], [-1.5448098182678223, -1.8903523683547974, -0.3240675628185272, 0.4394788146018982, -0.2020328938961029, -1.1875476837158203, -1.532731056213379, -3.2022669315338135, 0.21325477957725525, 1.2541725635528564, -1.1129212379455566, -0.7190403938293457, 0.9079806804656982, 1.4712870121002197, -0.6355549693107605, 0.22352930903434753, 1.3432062864303589, 0.32979971170425415, -0.9397370219230652, -0.6486190557479858, -0.3704130947589874, -1.710178256034851, 0.02936743199825287, 0.7806272506713867, 0.8192315697669983, -1.1879661083221436, -1.0907331705093384, 1.5151723623275757, -0.05845663323998451, 1.2482177019119263, 0.4029638171195984, -1.0394361019134521, -0.3215634226799011, -1.4530264139175415, 1.1990573406219482, 0.08140785992145538, -1.520623803138733, -0.26934385299682617, 0.3129677176475525, 1.6168092489242554, 0.720249593257904, 0.9066010117530823, 1.030151605606079, 0.12412616610527039, -0.629589855670929, 0.20434485375881195, -0.586924135684967, 0.4685344994068146, -1.3896420001983643, -0.679347038269043, 1.2057838439941406, 1.4095947742462158, 1.3307408094406128, -1.2705700397491455, -0.4410746395587921, 1.078856110572815, 0.6516510844230652, 0.5525809526443481, -1.0736479759216309, -0.07005034387111664, 0.5756737589836121, -0.010277518071234226, 0.10454341024160385, -0.5713696479797363], [-0.5766510367393494, -0.23365581035614014, 0.30733180046081543, -1.3557370901107788, -0.5864431858062744, -0.6131919026374817, 0.4106733798980713, 0.5954878330230713, 2.016080856323242, 1.134111762046814, 0.9183825850486755, -0.2035459280014038, -1.0760103464126587, 0.5905327796936035, -1.884705662727356, -2.3665153980255127, 0.9491751194000244, -0.988788366317749, 0.5368108153343201, 0.7833015322685242, 0.6951667070388794, -0.3892301619052887, 0.4453411102294922, -1.1872639656066895, -2.371880531311035, 0.7581377625465393, -1.2349357604980469, -2.037287473678589, -0.7732735872268677, 0.22795534133911133, -1.2339820861816406, 0.494327574968338, 0.19263646006584167, 1.4846632480621338, 1.695945382118225, -0.12017359584569931, 2.806630849838257, 0.439605712890625, 0.13685673475265503, 0.7276135087013245, 0.25386112928390503, 1.9020802974700928, -0.25033774971961975, -0.41375532746315, -1.9121034145355225, -0.15754592418670654, -0.2943529486656189, -0.6664856672286987, 0.580924928188324, -0.00983637385070324, 0.20736277103424072, 0.4221804141998291, -1.296589970588684, -1.2599081993103027, 1.0244981050491333, -0.36730316281318665, -0.8178373575210571, 1.5675989389419556, -1.57894766330719, 1.5197837352752686, -0.9730510711669922, -1.1294021606445312, -0.9392070770263672, 0.033203016966581345], [-0.5677086710929871, 1.1542853116989136, 0.7490747570991516, 0.16472794115543365, -0.5398998856544495, -0.7283636331558228, 0.46620801091194153, -0.5093337893486023, 0.21885544061660767, 0.2099328488111496, 0.07496664673089981, -0.39914149045944214, 2.5332019329071045, 1.169711947441101, -0.3405890166759491, 0.44006308913230896, -0.05610853061079979, -0.0651169940829277, -0.10915666818618774, 0.021722303703427315, -1.659155249595642, 0.018057042732834816, 0.7000495791435242, 1.4311140775680542, 0.4222228229045868, 0.43758049607276917, 0.5177309513092041, -0.19297462701797485, -2.8026680946350098, 0.4908570945262909, 1.1222257614135742, -1.245360255241394, 0.7402210235595703, 1.1023454666137695, 0.6448462009429932, -1.5978056192398071, 0.10397623479366302, 0.960789680480957, 0.7739921808242798, -1.9400213956832886, 0.3308555781841278, -0.49135473370552063, 0.48156481981277466, -1.326103925704956, -0.9086594581604004, 0.4512097239494324, 1.2566649913787842, 1.4287402629852295, -1.0693765878677368, 1.2877877950668335, -0.7637037038803101, -1.2661948204040527, -0.5292998552322388, -1.4574116468429565, 1.2172735929489136, 0.02383195050060749, 0.1368715763092041, -0.4588366150856018, 0.050781261175870895, 0.6555081605911255, 1.2568358182907104, -0.48153242468833923, 0.7462404370307922, 1.0504732131958008], [-0.8339444398880005, -1.1191694736480713, 1.087247371673584, 0.9804612398147583, 0.4733324348926544, -1.2908052206039429, 1.0374462604522705, -0.367897093296051, 2.3564188480377197, 1.1319061517715454, 0.8394847512245178, -0.34165018796920776, 1.5932258367538452, -1.0339306592941284, 0.45118314027786255, 0.963511049747467, -0.7144301533699036, 0.6808589100837708, 0.0764663964509964, -0.3427608013153076, 0.9867790341377258, -0.3533935546875, -0.08465756475925446, 0.7714104652404785, 0.5963046550750732, -2.101182460784912, 0.7517101168632507, -0.09389698505401611, -0.7088930606842041, -0.6698192954063416, 0.2923140525817871, 1.269089698791504, -0.560867428779602, -0.1575133204460144, 0.7735629081726074, -0.6276019215583801, 1.2672886848449707, -1.2159347534179688, 0.46397390961647034, 0.12087124586105347, -1.4640531539916992, -0.20786245167255402, -0.19706706702709198, 0.4150560200214386, -1.006888508796692, -0.04405154660344124, 0.0004614981298800558, -3.059603691101074, 0.41438204050064087, -0.37848520278930664, 2.550334930419922, -0.5125922560691833, 0.38519832491874695, -0.749421238899231, 0.48082929849624634, -0.046639155596494675, 0.9744066596031189, -2.213520050048828, 0.3899860978126526, -0.973019003868103, -0.3078223764896393, -2.7056612968444824, 1.0716317892074585, -0.15168523788452148], [-1.5940459966659546, 0.3751620352268219, -0.24717174470424652, 1.2115857601165771, -1.5586645603179932, -0.4830400347709656, 1.961434245109558, -0.9934710264205933, 0.8996787071228027, -0.017254697158932686, 1.1085134744644165, -0.8466002941131592, 1.6658964157104492, 1.6145118474960327, -0.5162152051925659, -1.0428096055984497, 0.9004082679748535, -0.66263747215271, 1.8103243112564087, -0.9733086824417114, -1.197584867477417, 0.08936359733343124, -0.6052403450012207, 1.710025668144226, -0.612510085105896, -1.017821192741394, -2.528620958328247, 0.5148882269859314, 1.6132694482803345, 0.10149148106575012, 0.4939407408237457, -0.6736893057823181, -2.33664608001709, -0.8729221224784851, -1.0941437482833862, 0.6086995005607605, 2.4479565620422363, -1.028203010559082, -1.0219630002975464, -0.2115272432565689, 1.4223103523254395, -0.2472166270017624, -0.6946368217468262, -0.1897415667772293, -0.07685001939535141, 0.48058614134788513, -0.608790397644043, -1.6589205265045166, 0.8683980703353882, 0.41722139716148376, -1.1444016695022583, 0.7051461935043335, -0.824799656867981, 0.4930937886238098, -0.2849169671535492, 0.2472720891237259, -0.7632278203964233, 1.0152777433395386, -0.12146203964948654, 1.1269606351852417, 0.8334763050079346, 0.2703242003917694, -1.601942539215088, -0.07932813465595245], [1.3301054239273071, 0.588828444480896, 1.2536386251449585, 1.2843480110168457, 0.767523467540741, -0.9054650068283081, 0.5710537433624268, -0.9154847860336304, -0.0951480120420456, 0.5366368889808655, 1.4327340126037598, -1.0774282217025757, 0.08662698417901993, -0.13660450279712677, -1.681861162185669, 0.39885440468788147, -0.3513592481613159, -1.236742377281189, -1.186712622642517, -0.0851144790649414, 0.6848024129867554, -1.3119574785232544, 1.0509626865386963, -0.6587303280830383, 1.732799768447876, -1.9787616729736328, -0.9048617482185364, 2.1060922145843506, 0.35144612193107605, -1.1485546827316284, 0.7082825303077698, 0.8237384557723999, -0.270404189825058, 0.9351756572723389, -1.0856174230575562, -1.2909915447235107, -1.7707445621490479, 1.4161044359207153, -0.03898772969841957, -1.4047255516052246, 1.3783433437347412, -0.01966238208115101, -0.58665531873703, 0.0132057536393404, -0.112839475274086, 0.1261897087097168, 0.6550498008728027, -0.9622849225997925, 1.25644052028656, -0.9160484075546265, 0.36313408613204956, -0.49295762181282043, 0.5107976198196411, 0.035212721675634384, -2.391832113265991, 1.8141353130340576, 0.47463473677635193, -0.23986609280109406, -0.6030665636062622, -0.28646034002304077, 0.005031818524003029, -0.957669198513031, 2.088347911834717, 0.11954587697982788], [-1.0493136644363403, 0.2217787206172943, -0.9972518682479858, 0.2728661596775055, -0.2828660011291504, 0.18275347352027893, 1.0891667604446411, -2.486290693283081, 1.0559958219528198, 1.3715934753417969, -0.8682126402854919, 1.3113808631896973, 0.6201081275939941, 2.20888090133667, 0.7052091360092163, 0.5269511342048645, -1.5080435276031494, 1.048590064048767, 0.3309668004512787, 0.5891819596290588, 0.48774006962776184, 0.3056950271129608, 0.5430200695991516, 0.3286319375038147, -1.1606754064559937, 0.7787877917289734, 0.9709652662277222, 0.4449506103992462, 1.617488980293274, 1.5697686672210693, -0.3389018177986145, 0.8482444286346436, -0.935841977596283, -0.817336916923523, 0.33813098073005676, 0.8967409729957581, 1.0725771188735962, 0.7112638354301453, -0.5142072439193726, -1.6172600984573364, -0.004149492364376783, 1.293318510055542, -0.33403414487838745, 0.3500401973724365, 0.7876706123352051, -0.35179245471954346, -1.6286109685897827, -0.4179877042770386, 2.186368703842163, 0.4438193142414093, -0.3573738932609558, -0.9882108569145203, 1.8139221668243408, -1.5718059539794922, -0.38595113158226013, 1.5286905765533447, -0.035216424614191055, 0.14596045017242432, -0.6043676733970642, 2.0801522731781006, 2.434314012527466, -1.968597412109375, -0.6342426538467407, 0.23553313314914703], [-1.9653358459472656, 0.07533111423254013, 0.33360180258750916, -0.6577147245407104, -1.0326744318008423, 1.2894291877746582, -1.0295003652572632, 0.8640609383583069, -0.5555505156517029, -0.5909048318862915, 1.7077152729034424, -0.3901897072792053, -0.08161615580320358, -1.8033554553985596, 0.5125851035118103, 2.765035629272461, -0.4394746720790863, 0.2710557281970978, 1.2231255769729614, -1.2863752841949463, -1.2273775339126587, 1.54231595993042, -0.5145843625068665, -1.4252724647521973, 0.5583769083023071, 0.460899293422699, -1.6587750911712646, -0.010549129918217659, 0.24742570519447327, -1.2899831533432007, 0.6103981137275696, 0.0072000701911747456, -1.2242788076400757, -1.4840654134750366, 0.16969989240169525, 0.005121850408613682, -0.35254037380218506, -0.7686124444007874, 1.3363193273544312, 1.188378095626831, -0.5034787654876709, 0.567570149898529, 0.425304114818573, 0.4510404169559479, 0.6386160850524902, 1.3617130517959595, 0.03610412776470184, 0.911040723323822, -1.6531332731246948, 0.24526190757751465, 1.2333035469055176, -1.2119474411010742, -0.24612033367156982, 0.3599153161048889, -1.1065897941589355, 1.1545186042785645, -0.1693509817123413, -0.9699727296829224, 0.6176232099533081, -1.5308207273483276, 0.9215635061264038, -0.989558756351471, -0.7963813543319702, 0.27541759610176086], [-0.642185628414154, 1.0150648355484009, -1.4735713005065918, -0.032451238483190536, -0.9976270198822021, 0.7145533561706543, 0.31195521354675293, -0.15151427686214447, 0.10560906678438187, -0.29754728078842163, -0.49186277389526367, -0.12698601186275482, 0.10477001219987869, -1.1623570919036865, -1.7132750749588013, -1.8309216499328613, 1.4483864307403564, 0.18449567258358002, 0.838626503944397, 1.2548273801803589, -0.8912302851676941, 1.9602900743484497, -1.0053139925003052, -0.9866381287574768, 2.271855592727661, -0.2852196991443634, -1.0916880369186401, 0.6813555955886841, -0.495295912027359, 0.8572039604187012, -0.5732235908508301, -2.1241579055786133, 0.08300238102674484, -1.0665016174316406, -0.4569104015827179, 0.21173737943172455, 0.020934339612722397, -0.6375190615653992, -1.312796711921692, -0.010180759243667126, -0.22233255207538605, -0.765638530254364, 0.963920533657074, -0.43106356263160706, 1.9326457977294922, -0.029992172494530678, -0.9221397638320923, 0.4410887062549591, 0.7033425569534302, 0.40494468808174133, 0.06074712797999382, -1.5959320068359375, -0.8510790467262268, -0.18385155498981476, 0.3952828049659729, 0.17151695489883423, -0.48952311277389526, 0.064651258289814, 0.00109372497536242, -1.0045149326324463, -1.3456904888153076, 0.9034464955329895, 0.8012663125991821, -0.887376606464386], [-1.1857103109359741, -0.42226922512054443, 0.9071964621543884, -0.6563761234283447, 0.03289227932691574, -0.5418165922164917, 2.53706693649292, -0.1394810825586319, 0.1642366498708725, -1.099259376525879, 1.1206719875335693, 1.1015242338180542, -0.361459344625473, -1.0657230615615845, 0.5021097660064697, 0.15283983945846558, -0.3795485198497772, -0.7282658815383911, -0.38961464166641235, -0.08522508293390274, -0.006703068036586046, 0.7818739414215088, -0.9337698221206665, -0.30814337730407715, 0.513877809047699, -0.7737793922424316, -0.6834427118301392, 0.36070406436920166, 0.31193238496780396, 0.2638089656829834, 1.6120069026947021, -0.6034941673278809, 0.08979414403438568, -1.876890778541565, 0.040993768721818924, -1.1075800657272339, 0.14062154293060303, 2.093628168106079, -1.1284517049789429, -0.37713077664375305, -0.07872230559587479, -1.0501736402511597, -0.8093673586845398, 1.409549355506897, 0.28924453258514404, 0.5351253747940063, 0.7039197087287903, 0.7498279809951782, 0.25214695930480957, 0.7673364877700806, 2.982537031173706, -1.1250956058502197, -1.3562567234039307, 0.7783613801002502, 0.19574084877967834, -1.2580851316452026, -1.1314588785171509, -1.384925365447998, -0.6163638830184937, -0.21578377485275269, -0.6245600581169128, -0.04269219562411308, 1.7241231203079224, 1.404741883277893], [-0.20560981333255768, 0.48329442739486694, 1.4091078042984009, 0.2284272164106369, 0.428671270608902, -0.16522730886936188, -0.6386964321136475, 0.09600493311882019, -1.480291724205017, -0.30836039781570435, 0.9531843662261963, -0.04218326508998871, -1.4911710023880005, -0.6409039497375488, -0.6355385780334473, 0.37013134360313416, -0.7586276531219482, 0.23753008246421814, 0.8144329786300659, 0.028740910813212395, -0.7790381908416748, -2.109347105026245, -0.2260996550321579, -0.25320708751678467, 1.0409940481185913, 0.16671346127986908, 2.160801649093628, -0.12284304946660995, -0.23326542973518372, -0.9345666766166687, 0.18092302978038788, 0.1670754998922348, -0.7342689633369446, 0.4172222912311554, 0.4878748953342438, -2.137625217437744, -0.41742828488349915, 0.2954343259334564, 0.602860689163208, -0.8754172921180725, 0.17364591360092163, 0.6765857338905334, -0.32215869426727295, 0.4157392382621765, -0.9629486203193665, 0.12063705176115036, 0.22196820378303528, 0.2527463436126709, -0.2942676246166229, 1.8503745794296265, -1.844567060470581, -0.6163901090621948, -0.9626634120941162, 0.3176649808883667, 2.035115957260132, -0.3889659345149994, 0.8045887351036072, 0.07963398844003677, 0.055923718959093094, -0.1180766150355339, 0.8743804097175598, -1.749285101890564, 1.447810411453247, 0.8584316968917847], [-0.8428683876991272, 0.38150718808174133, 0.4630703330039978, -0.2565574049949646, -1.887656569480896, 0.9711136817932129, -0.6597012281417847, 0.3200926184654236, 0.15449164807796478, 1.4700236320495605, 1.119690179824829, -0.45616772770881653, -0.39783212542533875, 2.026785135269165, -0.48240193724632263, 0.12254296988248825, -1.0951241254806519, 1.3173015117645264, -0.761620044708252, -0.06043105944991112, -1.22933030128479, 1.7732858657836914, 1.2388007640838623, -0.37698739767074585, 0.8156066536903381, 0.52801114320755, 0.038017015904188156, 0.6497460603713989, -0.8722893595695496, 1.6286849975585938, 1.4270118474960327, -1.3107450008392334, 0.7668609619140625, -1.1534595489501953, -0.4223928153514862, -0.2739517390727997, -0.2945464551448822, -0.6389978528022766, 0.41803786158561707, -0.8998284935951233, -0.11245296150445938, 0.9244562983512878, 1.2770578861236572, 2.2293028831481934, -0.8016759157180786, -0.610475480556488, 0.45695731043815613, 1.211378812789917, 1.0625028610229492, -0.8279008865356445, 0.19336409866809845, -0.8697975873947144, -0.6629904508590698, -0.4029827415943146, 2.1965482234954834, -1.6519372463226318, -0.7055208683013916, -0.7760434746742249, -0.203020378947258, -0.6559638381004333, -0.8694558143615723, -1.4298689365386963, 1.1728593111038208, -0.5173197388648987], [0.1578018218278885, 1.0375547409057617, 1.731652021408081, 1.7144232988357544, -0.9303849339485168, -1.1182172298431396, 0.02796652354300022, -0.7428575158119202, 0.29015159606933594, -0.9752257466316223, 0.05170163884758949, -0.25039783120155334, -2.1735358238220215, 1.1045401096343994, -0.23641036450862885, -0.3526574969291687, -0.14439652860164642, 0.8978134393692017, -2.6028501987457275, -0.9743494391441345, -0.3263442814350128, 0.7025544047355652, 0.10015633702278137, -0.4185923635959625, -0.3917909562587738, 0.12063264101743698, -1.2009726762771606, 1.0018025636672974, -0.39400771260261536, -0.4056638777256012, -0.334438681602478, -1.846906304359436, -0.8552013039588928, -0.8547144532203674, -0.2827088236808777, -1.8322399854660034, -0.5533458590507507, 1.279159426689148, 0.6764899492263794, 0.39984193444252014, -0.6183034181594849, 0.5721962451934814, -1.2503507137298584, 0.5152288675308228, -0.4536820650100708, -0.10680372267961502, 1.425018310546875, 1.3440775871276855, -1.716856837272644, -1.05970299243927, 1.2059890031814575, 0.9969700574874878, 1.0909873247146606, -0.6752135157585144, -0.9504467248916626, 0.8028871417045593, 0.8737478852272034, -0.35728874802589417, -0.9155246615409851, -4.0715012550354, 0.7346724271774292, 0.5288940072059631, 0.26940855383872986, 0.6963669061660767], [-0.14223866164684296, 0.21087798476219177, 0.27261096239089966, 0.8914791345596313, -1.4067013263702393, -2.879305839538574, -1.0315254926681519, 1.5645984411239624, -0.5951759815216064, 1.8566923141479492, -0.3526643216609955, -2.8576152324676514, -0.2729259431362152, -0.6543924808502197, -0.9917996525764465, 0.4627346694469452, 0.4160783886909485, 0.31669047474861145, -1.8217250108718872, 0.36677977442741394, 0.49262386560440063, -0.599096417427063, 1.6887009143829346, -0.8484071493148804, 0.49412304162979126, -0.06895439326763153, -0.24663269519805908, -0.01325974427163601, 2.4908487796783447, 0.22402065992355347, -0.22324872016906738, 1.9417306184768677, 1.397316813468933, 1.687916874885559, 1.7348384857177734, 0.583779513835907, 0.250620573759079, -0.702749490737915, 1.1252872943878174, 1.1286020278930664, 0.9296572208404541, 0.8717614412307739, 0.5458253026008606, -0.7421224117279053, -0.553283154964447, 1.650673508644104, -0.05901547893881798, -0.8962793350219727, -0.3025267422199249, 0.40337038040161133, 1.1210840940475464, -0.5450040698051453, 1.091766595840454, -0.5074588656425476, -0.482639878988266, -1.554256796836853, -0.3785460889339447, -1.2137726545333862, 0.24292530119419098, 0.11379308998584747, -1.9022538661956787, -1.593874454498291, 0.01276837382465601, 1.7522538900375366], [-0.2163982093334198, 0.08097297698259354, 1.4419915676116943, 1.4449687004089355, -1.5059101581573486, -1.1109516620635986, 2.4418017864227295, -0.19772371649742126, 0.21105659008026123, -0.10990971326828003, -1.3242989778518677, -0.3298802077770233, 1.8480640649795532, 0.7901648879051208, -0.2042338252067566, -0.8222563862800598, -0.8060484528541565, -0.4641941487789154, -0.5576266646385193, 0.4514048993587494, 0.4300365447998047, -0.4740208089351654, -0.8509902358055115, 0.7594257593154907, -0.47719353437423706, 0.2425270974636078, -1.230600357055664, -1.5101865530014038, 1.162400245666504, -0.5376050472259521, -1.025384545326233, -1.6881436109542847, 0.8197461366653442, -1.6449182033538818, 0.2975805699825287, 1.4563536643981934, 0.011916824616491795, 0.2581728398799896, -1.1348143815994263, 0.23803676664829254, 0.01600956730544567, 0.3234483301639557, -1.6062862873077393, -1.371158480644226, -0.9727422595024109, -0.8469953536987305, -1.7433063983917236, -0.5953215956687927, 0.5788148641586304, -1.1458215713500977, -1.685914397239685, 1.45452082157135, -0.3934752643108368, 1.0380489826202393, -0.5581467151641846, 0.7103900909423828, -1.1223045587539673, -1.1075117588043213, 0.3773932456970215, 1.6064319610595703, -0.3130265474319458, 0.26454973220825195, 2.198214292526245, -1.1036534309387207], [0.2882443368434906, -0.5170360207557678, 0.5081555843353271, 0.8682882189750671, 0.741237998008728, 0.7046788930892944, 0.5195220708847046, 1.3025548458099365, 0.48700955510139465, 0.28512606024742126, 0.06129666417837143, -0.6442514657974243, 0.4233010411262512, -0.28774726390838623, 0.5329652428627014, -1.4942954778671265, 1.5688958168029785, -0.5530461668968201, 0.5775259137153625, -0.7938653230667114, -0.22665609419345856, 0.20072148740291595, -0.3204496204853058, -1.0729472637176514, -0.9531349539756775, 0.24817782640457153, 0.45594674348831177, 0.03907677158713341, -0.7472138404846191, -1.8016235828399658, -1.4573458433151245, -0.9152376651763916, 0.9837046265602112, -0.6612880229949951, -1.1847907304763794, 0.9224777817726135, 0.9931661486625671, -0.334224671125412, -0.8102740049362183, 0.529702365398407, -1.218621015548706, 0.05276573449373245, 1.1009490489959717, -1.2311246395111084, -1.0153372287750244, 0.2571530342102051, -0.769565224647522, -0.279043972492218, -1.1323089599609375, -1.5768271684646606, 0.9383107423782349, -1.5870182514190674, -1.0474228858947754, -1.2722615003585815, -0.05486220866441727, 1.7630300521850586, -1.9497770071029663, 0.0427386574447155, 0.8044068813323975, 1.4395924806594849, -0.17027100920677185, 0.8179776668548584, -1.5774918794631958, -0.04986601322889328], [0.3083387315273285, 0.048097435384988785, -1.2835187911987305, 0.41154855489730835, -1.6383007764816284, 2.4563677310943604, -0.1360326111316681, 1.6031558513641357, 0.5072651505470276, 1.3956102132797241, -0.23075172305107117, -1.0785892009735107, -0.6680383682250977, -0.7210286259651184, -0.30915990471839905, -0.35537829995155334, -1.2664270401000977, -0.20711365342140198, -0.6295702457427979, -0.21432392299175262, -1.2044410705566406, 2.19038462638855, 1.6073298454284668, -0.4907554090023041, -0.3161522448062897, -0.1898542046546936, 1.606075406074524, -0.07359375804662704, 1.020959496498108, 0.5366296172142029, 0.224817156791687, -0.9780979156494141, 1.3605791330337524, 1.0123984813690186, -1.159034252166748, -0.061734843999147415, 0.9055694937705994, 0.5002928972244263, -0.9005448222160339, -1.3255757093429565, 0.5608910322189331, 2.158336877822876, 2.1849253177642822, 0.5533936619758606, -0.558451235294342, 0.6195804476737976, 0.0012554503045976162, -1.7945760488510132, 1.6140704154968262, 0.43386149406433105, -1.5997843742370605, -0.23654627799987793, -0.7640473246574402, 0.5670530796051025, 1.004548192024231, 1.2806096076965332, 0.3529197573661804, 1.2144018411636353, 1.0025076866149902, -1.0343568325042725, 0.023677121847867966, 1.9642809629440308, 0.05208561196923256, -1.139278769493103], [1.538803219795227, -0.7424434423446655, -0.5886244177818298, -0.2940945327281952, 0.5502108931541443, 1.0494489669799805, -0.006359168794006109, -0.9573516845703125, 0.006949369329959154, -1.6762027740478516, -0.00391369080170989, -1.235965609550476, -0.9192748069763184, -1.8294703960418701, -0.20918667316436768, 0.7590420246124268, 1.1070337295532227, -1.2910577058792114, -2.0125744342803955, -1.901315689086914, -1.429776906967163, 0.3805255889892578, 1.0384197235107422, 0.13265866041183472, 0.14343442022800446, -1.8486509323120117, -0.004091027658432722, -0.5484753847122192, -0.4033908247947693, -1.171532392501831, 1.1163239479064941, -0.1355394721031189, 0.22769111394882202, 1.2627753019332886, 0.8209423422813416, -1.7082973718643188, 0.3740082085132599, 0.6072905659675598, -1.60298752784729, -0.12933629751205444, 1.7680680751800537, 0.685924232006073, -0.30799296498298645, -1.5128400325775146, -0.7158718705177307, -0.7528753876686096, -0.08541686832904816, -2.712690830230713, -0.5137230157852173, -0.09028107672929764, -1.377495288848877, -0.22615021467208862, 1.3910884857177734, 0.39180824160575867, 0.6135798096656799, -1.4400253295898438, -1.1771824359893799, 1.3170212507247925, 0.041318878531455994, 0.5724881291389465, -0.5520543456077576, 1.8390752077102661, 0.1338915079832077, -0.24223089218139648], [-1.0461491346359253, -1.0365663766860962, 1.367232322692871, 0.4385814368724823, 0.7027334570884705, -1.157881259918213, 1.5923410654067993, 2.1377675533294678, 1.719097375869751, -0.8292967677116394, -1.0284719467163086, 0.6114575862884521, 0.6911693215370178, 0.6354750990867615, -1.169012188911438, 1.531540870666504, 1.7431073188781738, -0.7172178626060486, -1.0863245725631714, 0.21133765578269958, 0.6164808869361877, 0.2618925869464874, 0.7774861454963684, 0.10508394241333008, 1.34011971950531, -0.9201001524925232, -1.3023945093154907, 1.0997599363327026, 0.2691688537597656, -0.7078616619110107, 0.6746894717216492, 2.4146671295166016, 0.633899986743927, -1.1133441925048828, -0.9688613414764404, -0.6953241229057312, -0.48108452558517456, 0.669012725353241, 0.9513556361198425, -0.9292692542076111, 0.6131798028945923, -0.857704222202301, 0.7235234975814819, -0.5687964558601379, 0.0034108019899576902, -0.8809182643890381, -0.6853416562080383, -1.6399216651916504, -1.9661715030670166, 1.447402000427246, -0.3242699205875397, 0.053527310490608215, 0.030653465539216995, 0.41198137402534485, 0.6065507531166077, 1.826367974281311, -1.364123821258545, 1.4998656511306763, 0.6265401840209961, -1.2559760808944702, 0.6200955510139465, -0.32220256328582764, 1.9251071214675903, -1.3947147130966187], [0.8813693523406982, -0.15663379430770874, -0.7974188327789307, -1.5847806930541992, -0.12247329950332642, 0.33140257000923157, 0.24553900957107544, 0.05321548506617546, 0.485576868057251, 1.1817924976348877, 1.503049612045288, 0.16658265888690948, 1.0718352794647217, 0.21489264070987701, 0.4294792115688324, 1.2764836549758911, -1.948194146156311, -0.17747704684734344, -0.24268753826618195, 0.9862154126167297, -0.9459299445152283, 0.8098989725112915, 0.5597450733184814, 0.4347982108592987, -0.3556179702281952, -0.7362256646156311, 1.0549569129943848, -0.12911827862262726, 1.6404564380645752, -0.9323194622993469, 1.2451978921890259, -2.0620832443237305, -0.17040829360485077, -1.183579444885254, 0.6435643434524536, 1.3088877201080322, 0.017948707565665245, -0.3637235760688782, -0.6592141389846802, 0.6846370100975037, 0.6694515943527222, -0.10859449952840805, -1.138773798942566, -0.5493617653846741, -0.3745875060558319, -0.38421693444252014, -0.40515995025634766, -2.1026804447174072, 1.251739263534546, -0.4608892500400543, 1.6625053882598877, -0.0933714509010315, -1.7363944053649902, 0.3320324122905731, -0.3278796672821045, 0.1585775464773178, 1.9029983282089233, -1.3166284561157227, -0.7302414774894714, 2.373727321624756, -0.3170437812805176, -0.5568709373474121, 0.13413994014263153, -0.7095860838890076], [-0.10519517213106155, -1.2064154148101807, -1.1299617290496826, -0.5339111089706421, 1.4850389957427979, -0.32668617367744446, 0.13111841678619385, -0.4238561689853668, 0.7276294827461243, -3.4874861240386963, -0.6929120421409607, 0.46143171191215515, 0.17565639317035675, 0.6540634036064148, -0.45051389932632446, 2.0885536670684814, 1.5601792335510254, -0.34603604674339294, -0.9454497694969177, -0.28742000460624695, 2.0566372871398926, 0.6021454930305481, 0.8026131987571716, -2.23826003074646, -0.1392015516757965, -0.3373952805995941, -0.6736832857131958, 0.1272961050271988, 1.622067928314209, 1.3086211681365967, 0.27383553981781006, 0.2260953038930893, 1.0266703367233276, 0.6096252799034119, -2.294257879257202, -1.1599524021148682, -0.5722787976264954, -1.5522981882095337, 0.2502758800983429, -1.9682968854904175, -2.0227930545806885, -0.8801015019416809, -0.6522613763809204, -1.1476646661758423, 0.5720983147621155, 0.08599843084812164, -2.2013838291168213, -0.12441825866699219, -0.4025159180164337, 0.2844999432563782, -0.34490305185317993, 0.2821658253669739, 0.2821871042251587, -1.1568180322647095, 0.49294355511665344, 0.8922459483146667, -0.8727523684501648, -0.23460933566093445, -2.0458126068115234, -0.9228861331939697, -1.0037026405334473, 1.1431788206100464, -1.5578103065490723, 1.3956488370895386], [-1.1504218578338623, 0.17653711140155792, 1.5287973880767822, 0.8925771117210388, 0.6638772487640381, 0.782755970954895, -1.636728286743164, 0.16255247592926025, 1.9653654098510742, 0.27819615602493286, 1.2743326425552368, 0.493136465549469, -1.527708888053894, 0.03835504129528999, 0.6467341780662537, 0.7369523048400879, 0.24938422441482544, 0.42214494943618774, -1.535683035850525, 1.6603899002075195, 0.6916346549987793, -1.0392024517059326, -0.6293090581893921, 0.7747747898101807, -0.08334669470787048, 0.7338200211524963, -0.9844276309013367, -0.5168019533157349, -0.12374367564916611, 1.2572135925292969, 0.24139180779457092, 1.5201809406280518, 1.3647228479385376, -1.8780841827392578, 2.203935384750366, 0.7198796272277832, -0.001062982715666294, -1.1465224027633667, -1.3031946420669556, 0.4131089448928833, -0.4460509419441223, -0.031771473586559296, -0.45339953899383545, 1.1449472904205322, 0.9237654209136963, 1.676865577697754, -0.4939849376678467, 0.26284274458885193, 0.6347408890724182, -0.44022226333618164, 0.9390334486961365, 0.07901330292224884, -0.12566570937633514, -0.21222639083862305, -1.1394885778427124, -1.836020827293396, 0.6807261109352112, -2.2293646335601807, 0.0071545313112437725, -0.8576619625091553, 0.25606992840766907, -1.7361363172531128, 1.1944338083267212, 0.6641572117805481], [-0.24644382297992706, 0.21654191613197327, 0.4212946891784668, 0.2206486314535141, -0.7474859952926636, 1.8404752016067505, 0.8884570598602295, -0.4109991788864136, -0.4816344678401947, 0.4584502875804901, 0.1115679144859314, -0.6900515556335449, 0.5418622493743896, -0.9982362985610962, -0.06996341794729233, -0.41167253255844116, 0.05440481752157211, 0.532535195350647, 0.1309807002544403, 0.5829412341117859, 1.5110118389129639, 3.264751434326172, 0.12084643542766571, 0.5724976658821106, -1.0012388229370117, -0.5561054944992065, -1.1930358409881592, -0.9353528022766113, 0.9539756774902344, 0.38193556666374207, 1.4649815559387207, -0.7930092215538025, -0.6666709780693054, 1.3372390270233154, 0.5082827806472778, -0.3127465546131134, 1.3432049751281738, 0.1022326722741127, 0.4303974509239197, 1.1320557594299316, 0.34746262431144714, 1.0074608325958252, -0.4887048602104187, 1.9052551984786987, -0.5083696246147156, -0.7709946632385254, -0.21767698228359222, 1.1543738842010498, -0.3861132562160492, 0.4791805148124695, 1.437809944152832, -0.3102080821990967, -2.4301559925079346, -1.3302932977676392, 1.3221744298934937, 0.3364551067352295, -0.46379581093788147, 0.13372497260570526, -0.34089961647987366, -1.1206690073013306, 1.2532875537872314, 1.2954659461975098, 0.3102419674396515, -1.6795176267623901], [-0.4072396755218506, 0.0831386148929596, 0.5827471613883972, 1.5036940574645996, -0.6565195918083191, 0.34703290462493896, -0.41080066561698914, -0.26503655314445496, -0.4606768488883972, -0.7437638640403748, -1.0101842880249023, 1.227128028869629, 0.13391457498073578, 0.49519410729408264, 1.3035608530044556, -0.10111425071954727, 0.4814782738685608, -0.7268916368484497, -0.3058202564716339, -0.6980425119400024, 0.06001795083284378, -0.04001376032829285, 0.11632316559553146, 0.4825737774372101, -0.02009507082402706, 0.15095791220664978, 0.5596843361854553, -0.018480191007256508, 2.359595775604248, 0.5352097153663635, 0.549765408039093, 0.46567389369010925, 0.05801556631922722, -1.098514199256897, 0.7671109437942505, 0.7423408031463623, 0.4825707972049713, -0.4019104242324829, 1.2010334730148315, 0.6519529819488525, 0.8832811713218689, 1.1388063430786133, 0.6327375173568726, 1.8034011125564575, 1.727118730545044, 0.851033091545105, 1.8257418870925903, 1.3918092250823975, -1.539368748664856, 0.2853911221027374, 2.083732843399048, 0.7713251709938049, -1.063809871673584, -0.6917838454246521, 0.16292385756969452, -0.15486812591552734, 0.7533217072486877, -2.090949773788452, 2.6308608055114746, 0.8806796073913574, -1.0039273500442505, 0.8616669774055481, -0.1687423586845398, 0.37425389885902405], [-1.3452308177947998, 0.22037020325660706, 0.003428652184084058, -1.2974753379821777, -0.4055270552635193, 0.7424523234367371, -1.4207756519317627, 0.7169353365898132, 0.4310981333255768, 0.24241021275520325, -0.9393451809883118, -0.9524199366569519, -1.0776780843734741, 0.3000709116458893, -0.37705743312835693, -0.578947126865387, 1.1465362310409546, -0.4413336217403412, -0.9403424859046936, 0.6736963391304016, -1.4179508686065674, 0.8397260904312134, 1.5086677074432373, -0.03485769405961037, 0.36068665981292725, -0.740168571472168, 0.930778980255127, 1.6378475427627563, -0.14525015652179718, -0.8863739967346191, 1.0956411361694336, -0.131266787648201, 0.12309170514345169, 1.7407093048095703, 1.6810357570648193, 0.39134547114372253, 0.43713900446891785, -2.0469117164611816, -0.5444062948226929, 0.40753409266471863, -0.38794830441474915, -0.7214906811714172, -0.48510226607322693, 1.2069895267486572, -0.1426560878753662, 0.6498965620994568, -0.05308711156249046, -1.434921383857727, 0.31852343678474426, 2.5082273483276367, -0.08284509181976318, -0.5045045614242554, 0.8547922372817993, 0.3016025722026825, 0.12986896932125092, -0.15018385648727417, -0.8542225956916809, -0.714266836643219, -1.0150070190429688, -0.9361046552658081, 0.02564527653157711, 0.13909851014614105, 0.471484899520874, -0.645389199256897], [-0.9658713340759277, -0.7697638869285583, 0.6324966549873352, -1.6111077070236206, 1.3883814811706543, -1.2245975732803345, -0.34016117453575134, 0.04886065423488617, -1.4309502840042114, -0.20578442513942719, -1.6626155376434326, 0.2128385454416275, 0.018469402566552162, 0.1564905196428299, 0.2915254533290863, 1.709121584892273, 0.8236929774284363, 0.24465222656726837, -1.0758455991744995, -0.5698959827423096, -1.133058786392212, -0.2973697781562805, 0.9723010659217834, -0.37848469614982605, -0.5804280638694763, 0.54475337266922, 0.5562041997909546, -1.404723882675171, -0.6842694282531738, 0.2525146007537842, 0.13235458731651306, -1.5408434867858887, -0.792113721370697, -0.10927383601665497, -0.18941950798034668, 1.0937225818634033, -1.5270440578460693, -0.7215797305107117, -1.1413944959640503, 0.13195128738880157, -0.08643680065870285, -0.1717311292886734, 0.9813408255577087, -1.1963108777999878, 0.22568462789058685, 0.2628366947174072, -0.43290576338768005, 0.609259307384491, -0.535322904586792, 1.121969223022461, -0.9093392491340637, 2.0725905895233154, 1.3299564123153687, -0.2513352632522583, -0.3295241892337799, -0.6617182493209839, 0.4937197268009186, 0.8131508231163025, -0.6172994375228882, 0.18933172523975372, -2.191837787628174, -0.2715349495410919, 0.8165827989578247, 0.3542436957359314], [0.3222901225090027, -0.8624201416969299, -1.7857496738433838, -0.7502837777137756, -0.19436651468276978, 0.13966456055641174, -1.1283810138702393, 0.7345193028450012, 0.32285815477371216, -1.2633700370788574, -0.722841203212738, 0.3721160292625427, 0.21939247846603394, -0.011698842979967594, -0.3726472556591034, 0.028593525290489197, 0.3008037209510803, -0.19911064207553864, -0.004257740452885628, 0.011178180575370789, -0.4916982650756836, -0.7370222210884094, 0.923538088798523, -0.9080973863601685, 0.1555895060300827, 1.651279091835022, -2.691929578781128, -0.33619874715805054, 0.9576000571250916, 2.702946662902832, 1.2225151062011719, 1.1691287755966187, -0.5567744374275208, -0.8386843800544739, 0.6273244023323059, 0.11226925998926163, 0.419941782951355, 1.3386873006820679, 1.3379589319229126, -1.12528395652771, -0.7785962224006653, 0.03492553159594536, 0.6199784874916077, -0.025626864284276962, 0.4389790892601013, -0.5186885595321655, 1.125429630279541, -1.1787902116775513, -0.18731997907161713, -1.4114892482757568, 0.06163826957345009, 0.39236924052238464, -1.325421929359436, -0.5299797058105469, 0.0043764314614236355, -0.795462965965271, 0.5591709017753601, 0.5754032731056213, 0.38620322942733765, 1.7461985349655151, -0.2571084201335907, 1.6667660474777222, 0.46372872591018677, -0.305536687374115], [0.8082391619682312, -0.14101877808570862, -2.1722898483276367, -1.8396918773651123, 1.2097452878952026, 1.0781258344650269, -0.7994107007980347, -2.080437183380127, -0.6769929528236389, 1.5570851564407349, 0.11708852648735046, 0.8060800433158875, -1.9581806659698486, -1.7266993522644043, 0.4514191448688507, 0.8565046787261963, 0.22587864100933075, 0.17656032741069794, 0.4195370376110077, -1.7032579183578491, 1.5001475811004639, -0.6227647066116333, -0.043796002864837646, 0.6133003830909729, 1.081396222114563, -0.6933985948562622, 0.7852603197097778, -0.4216617941856384, 1.021126627922058, -0.97849440574646, -0.7830737233161926, 0.007953372783958912, -0.9224460124969482, 0.5210887789726257, 1.2628132104873657, 0.4591262638568878, 1.4783838987350464, -0.5701932311058044, 1.5715755224227905, -0.24363335967063904, -0.48588573932647705, 0.14583268761634827, -1.0782535076141357, -0.8078874349594116, 0.4748324453830719, -0.7555316686630249, -0.27287644147872925, -1.1525769233703613, 0.0025770661886781454, 0.7758843302726746, -0.4767482280731201, 1.2106283903121948, 0.26472947001457214, 0.05225563049316406, -0.8825867772102356, -1.4241222143173218, -0.9286234974861145, -1.7854695320129395, 0.17429420351982117, -0.48517927527427673, 0.9919505715370178, -0.5460802316665649, -0.13741132616996765, -0.19474032521247864], [0.2633819878101349, 0.9261161088943481, 0.15785770118236542, -0.3929901719093323, -0.6514317989349365, 1.1243891716003418, -0.3354024887084961, 0.6631230115890503, -0.4907802939414978, 0.5395029783248901, 0.7829365730285645, -0.6197264790534973, 1.456478476524353, -0.45386841893196106, -1.0314123630523682, -0.7225104570388794, 0.041610781103372574, -0.31212863326072693, -0.4834439158439636, -1.416575312614441, 0.3681544065475464, 0.5964154601097107, 0.6229429841041565, 0.8672391176223755, -0.4852295517921448, -0.9874946475028992, 1.3030290603637695, -0.651532769203186, -1.5969585180282593, 0.023157546296715736, -0.9857431054115295, 3.0243380069732666, 0.17501474916934967, 0.543575644493103, 1.792248249053955, -0.2919500172138214, -0.910162091255188, -1.1159260272979736, -0.5557296276092529, 0.09585019946098328, 1.6757547855377197, -0.9505637288093567, -0.8874116539955139, 0.09744968265295029, 0.45948314666748047, 0.17018599808216095, 1.795562505722046, 0.7938768267631531, 0.6070528626441956, 0.4102615714073181, -0.24128836393356323, -1.3218777179718018, -1.5887538194656372, -0.8508902192115784, -0.41689011454582214, 0.8490691184997559, 0.8154087662696838, -0.17703013122081757, -0.5457451343536377, -1.828344464302063, -0.5981720685958862, -0.5659136176109314, 0.20625746250152588, -0.3909227252006531], [-1.4993678331375122, 2.4680755138397217, -0.0888582319021225, -0.4023051857948303, -1.544222116470337, -0.7046037316322327, 1.4491949081420898, -0.42990365624427795, 0.13233067095279694, -0.4784187078475952, 0.27643051743507385, 1.0417615175247192, 1.4316620826721191, 0.3472411334514618, 0.5378616452217102, -0.5375709533691406, -1.803217887878418, -0.23306909203529358, -0.14714543521404266, 0.14147426187992096, -1.0284945964813232, 0.16439712047576904, -0.28132715821266174, 0.7705664038658142, -0.21602773666381836, -1.5275150537490845, -0.9135640859603882, 0.4985659420490265, 2.3630528450012207, 0.6020649075508118, -0.7647257447242737, -0.868807852268219, 0.57663494348526, 0.051045484840869904, 1.1799705028533936, 2.2561302185058594, -1.1263915300369263, 0.6255389451980591, -0.6867384910583496, -1.3534873723983765, 0.5744875073432922, -0.33409538865089417, -0.6299471855163574, 0.9652613997459412, -0.7882949709892273, -1.582899570465088, 0.27542734146118164, -0.33838674426078796, -1.5601375102996826, -0.13280807435512543, -1.1133567094802856, 0.16379205882549286, -0.21155722439289093, 0.8120149970054626, -0.08091360330581665, 1.4127334356307983, -2.2412281036376953, -1.0196349620819092, -1.9542840719223022, 2.065972089767456, 1.820968747138977, 1.3304764032363892, -0.5519596934318542, -0.49491187930107117], [0.2120225727558136, 1.0291504859924316, -0.18780659139156342, 0.8164224624633789, -1.5329631567001343, 0.43870770931243896, -0.41397178173065186, 0.315485417842865, 1.716981053352356, -0.8498461246490479, -0.026904651895165443, 1.6526581048965454, -1.1248582601547241, 0.6532202959060669, 1.415431022644043, -1.3777638673782349, 0.030224665999412537, -0.8685106635093689, 1.2314423322677612, -0.9225368499755859, -1.6043422222137451, 1.0604689121246338, -1.1681315898895264, -0.15961790084838867, 1.2414734363555908, -1.107362985610962, 0.9213684797286987, -0.703117847442627, -0.07052169740200043, 0.7395895719528198, -1.0782315731048584, 0.09741252660751343, -0.6585789918899536, -0.25865259766578674, 0.30008214712142944, 1.4604345560073853, -0.05454658344388008, 0.4159802496433258, -0.8498021960258484, 1.054945468902588, 1.4947539567947388, 1.6723188161849976, 1.2434568405151367, 0.2658187747001648, 0.34887757897377014, 2.8083643913269043, -1.5077351331710815, 0.7869940400123596, -0.07776113599538803, -0.7786396741867065, 0.5171452760696411, 0.5587510466575623, 0.7026286721229553, 0.24884353578090668, 0.49918949604034424, -0.4200703799724579, -0.5754551887512207, 0.2718240022659302, -0.3688465356826782, -2.1782021522521973, -1.1713379621505737, -1.0386706590652466, 0.37323179841041565, -2.4257144927978516], [-2.0340192317962646, -1.0726568698883057, 0.13308051228523254, -0.018635066226124763, -0.8340634703636169, -1.301112413406372, 0.6396077275276184, -1.496464729309082, 0.5386242270469666, 0.607347309589386, -1.1714190244674683, 0.022952163591980934, 0.8819953203201294, 1.3985483646392822, 0.404335081577301, -0.39509519934654236, 1.0733623504638672, -0.6598111987113953, -0.13636571168899536, -0.7951773405075073, 0.5735600590705872, -0.8904993534088135, -0.4417390525341034, -1.059144139289856, -2.0780446529388428, 0.5687474608421326, 0.22476576268672943, 0.02009928971529007, 0.3197455406188965, 0.08777909725904465, -0.5315096974372864, 0.2582587003707886, -0.9426541924476624, -0.347246378660202, 0.170057013630867, 0.26837876439094543, 1.7894868850708008, -1.041138768196106, -0.8700069785118103, -0.48090407252311707, 0.5679321885108948, 1.309190034866333, 1.4200949668884277, 0.030383825302124023, -1.3347697257995605, 1.6593017578125, -0.6079816818237305, -1.624219536781311, -0.4524902403354645, 0.494184672832489, -0.3828214406967163, -1.1001980304718018, 0.031014459207654, -0.15386301279067993, -0.7451674938201904, -0.2739010453224182, -2.2628142833709717, -0.88744056224823, -1.6074566841125488, -0.06660541892051697, -0.22145266830921173, -0.07804457098245621, 0.8839309811592102, -1.2924243211746216], [-0.12908770143985748, 0.3439213037490845, -1.3855371475219727, -0.891131579875946, -0.7804602384567261, -0.8816887736320496, 0.13729985058307648, -1.042109727859497, -0.8930457234382629, -0.4837459921836853, -0.050865184515714645, 0.9594318866729736, 0.35165607929229736, 1.5109567642211914, -2.4633734226226807, -0.21289046108722687, 0.5365811586380005, 0.13466155529022217, 0.699096143245697, -0.5173818469047546, -0.9307330250740051, -1.015995740890503, -0.4670332074165344, -0.8185445666313171, -0.9724526405334473, -2.1081902980804443, 0.1372295767068863, -0.3849065601825714, 0.25183430314064026, 1.483726143836975, 0.21981677412986755, 0.7761898636817932, 0.8380807638168335, -0.756338357925415, 0.3004404306411743, -1.2387140989303589, -0.7932409644126892, 0.37324321269989014, 2.071995496749878, -0.8612308502197266, 1.1874382495880127, -0.15307915210723877, -0.1365543156862259, -0.46276605129241943, -0.11149071902036667, -1.1468324661254883, 0.324013888835907, 0.25776082277297974, 1.2749526500701904, -0.6949681639671326, -1.8369196653366089, 0.5730980634689331, 0.4715292453765869, -0.603003978729248, 0.05230957642197609, -0.8912390470504761, 1.1661335229873657, 0.4857456088066101, 1.8375296592712402, 1.3959304094314575, 0.3632192015647888, 0.6446353197097778, -2.2596826553344727, 0.48865190148353577], [0.7694674134254456, -0.6324983239173889, -1.0638700723648071, 1.4995967149734497, -2.4038381576538086, -0.013627506792545319, -0.8057766556739807, 1.2755587100982666, -0.18727467954158783, -1.278719186782837, -1.5682448148727417, -0.3785272240638733, 0.7181723713874817, -1.2047823667526245, 0.004061362240463495, -1.2758976221084595, -0.7893633842468262, 0.600490391254425, 0.2983365058898926, 0.2938339114189148, -1.2495359182357788, -0.25792914628982544, 0.002866125199943781, -0.31326743960380554, 0.31939706206321716, -1.7653194665908813, 2.424612045288086, 0.5425469279289246, -0.7161868810653687, -0.5875435471534729, 0.029159223660826683, -1.0263992547988892, -0.09228046238422394, -0.17161685228347778, 1.439987301826477, 0.05114904046058655, 0.8968653082847595, -0.09818781167268753, 0.39963531494140625, 0.010980295948684216, 0.4761337637901306, -0.17414379119873047, -1.1512885093688965, -0.29976174235343933, -0.8621150851249695, 0.43384113907814026, 0.5239064693450928, 0.6794281005859375, -0.11449787765741348, 1.0952078104019165, 1.2388358116149902, -0.11784185469150543, -1.0535777807235718, 1.0022085905075073, -2.43363881111145, 0.6541638970375061, -1.177570104598999, -0.10308347642421722, 0.05617934837937355, 2.23012638092041, 0.1011471375823021, -0.8309860825538635, -0.9632083177566528, 0.011582341976463795], [0.673736035823822, 0.4034717082977295, -1.165183424949646, -2.680234432220459, 0.41217899322509766, 0.4401262402534485, 0.7099778056144714, 0.13082903623580933, -0.6266937851905823, 1.6492966413497925, 1.1501545906066895, -0.12154603749513626, 0.5415581464767456, 0.9423655271530151, 0.1247718408703804, -0.7843578457832336, -0.0441855788230896, -0.3902856409549713, -0.5398216843605042, 0.2761731743812561, 0.24972179532051086, 1.5631840229034424, -0.2163466215133667, 0.9849998950958252, 0.35819172859191895, -1.185524582862854, -0.3336503505706787, 0.7641428709030151, -0.1136975884437561, 0.2021738886833191, 1.0793795585632324, -0.6680211424827576, -0.35357511043548584, 0.1569085270166397, 0.11466474831104279, -0.21840235590934753, -0.41286414861679077, -1.3021069765090942, -0.12519550323486328, -0.927848219871521, -0.4441450536251068, -0.6624905467033386, -1.1500927209854126, 1.8965498208999634, -0.24534595012664795, 0.3589804172515869, 0.8881641626358032, -0.7461386322975159, -0.7898014783859253, 0.7460607290267944, -1.1806344985961914, -1.0596615076065063, -1.557847023010254, 1.3893582820892334, 0.1764451265335083, 0.2625890374183655, -0.996850848197937, 1.421139121055603, -0.5929448008537292, -0.213112935423851, -0.16161513328552246, 2.111248731613159, -0.2507763206958771, 2.1180179119110107], [0.6268508434295654, -0.5020327568054199, -0.706520140171051, -2.4197070598602295, -1.042176604270935, 0.2975875437259674, 0.2855328917503357, 0.6706562042236328, 1.271855115890503, 1.216166377067566, -1.5905736684799194, -0.8584953546524048, 0.0022122010122984648, 0.7775979042053223, 0.004376884549856186, 0.40312469005584717, 0.4571661055088043, 2.3586575984954834, -0.8496173620223999, -0.2592713534832001, -0.5509191751480103, -1.7220673561096191, -0.8096367120742798, 0.5268564820289612, -0.3791114091873169, 0.4643774926662445, 0.5045363306999207, -0.43855729699134827, -1.431283712387085, 0.806630551815033, -0.807088315486908, -1.4082881212234497, 0.5375077724456787, -0.2900632321834564, 1.5785164833068848, 1.2198528051376343, 0.4862573444843292, 2.3775084018707275, -0.07940457761287689, 0.09954581409692764, 1.166542649269104, 0.09392376989126205, 1.5565944910049438, -1.4236162900924683, 0.09860973805189133, 1.080092430114746, -1.7788668870925903, 0.14599277079105377, -0.26076558232307434, -0.3030620515346527, 1.194145917892456, 0.6589899063110352, 0.8494441509246826, 1.770445466041565, -0.22822579741477966, 0.5711764693260193, 0.873818039894104, -0.6544414758682251, 0.9384600520133972, -1.0482640266418457, 0.18725812435150146, 0.22226539254188538, 0.830496609210968, -1.0704554319381714], [-0.4695257246494293, -1.229367971420288, -0.6319196820259094, -0.819933295249939, 2.5599443912506104, -0.7396562099456787, -0.12683157622814178, 0.6737151741981506, 0.3709172308444977, 0.830707311630249, -1.033530831336975, -1.2042444944381714, 1.4837689399719238, 0.7895241975784302, 0.1080162301659584, -0.24489881098270416, 2.0110414028167725, -1.1142122745513916, 0.018428368493914604, 1.555760383605957, -0.18452157080173492, -0.4258638322353363, 0.8416683673858643, -0.13213473558425903, -2.7900636196136475, -1.0910155773162842, -0.302360475063324, 1.8281779289245605, -0.45555683970451355, -0.0043471939861774445, 0.325569212436676, -0.25922513008117676, -0.7417997121810913, 0.6110662221908569, -2.533850908279419, 1.8286921977996826, -0.6551199555397034, -1.3317927122116089, -0.6956321001052856, 0.7703297138214111, -0.9529100060462952, -0.8115208148956299, -0.5831015110015869, -0.6589857339859009, -0.984250545501709, -1.7489306926727295, -0.9372261762619019, 0.19400250911712646, 0.6878249645233154, 0.3829881548881531, -0.04654227942228317, -1.0100258588790894, -0.22958077490329742, -1.170412540435791, 0.958156168460846, 0.576184868812561, -0.8278855085372925, 1.3774480819702148, -1.0604232549667358, -2.9483084678649902, 0.25663232803344727, 0.015736250206828117, 1.639823079109192, 1.2114635705947876], [0.01798165962100029, -0.7673525214195251, 0.3371009826660156, 0.32237085700035095, -0.9534339308738708, 1.6842283010482788, -0.76068115234375, -0.44432342052459717, 0.850634753704071, 1.9299713373184204, -0.19917242228984833, 0.8089960217475891, 0.09221170842647552, -1.3570119142532349, -1.3006703853607178, 1.2889467477798462, 1.4117600917816162, -0.3116062581539154, 0.17810232937335968, 0.22561639547348022, -0.09319578856229782, -0.7237114906311035, -0.7477866411209106, 1.7018301486968994, -0.13386216759681702, 0.7615270614624023, 1.4412667751312256, -0.5788936614990234, -0.2557714581489563, 2.0036423206329346, 0.1307995617389679, -0.7782888412475586, 1.7707701921463013, 0.5215643048286438, 2.6157314777374268, -1.0146594047546387, 0.6217809319496155, 0.08406082540750504, -0.8518686294555664, 0.37832003831863403, 1.195702075958252, 1.2900062799453735, -0.09338752180337906, -0.4323309659957886, 1.4419902563095093, 0.46212366223335266, 0.5808435678482056, 1.0209064483642578, -1.2705745697021484, -0.5860698819160461, -0.9823832511901855, -0.7407710552215576, 1.207311987876892, 0.8965275883674622, 0.4205911457538605, 0.11328248679637909, -1.312928557395935, -0.575579047203064, -1.2697768211364746, 0.3272998631000519, 0.13549721240997314, -0.16771313548088074, 1.5407929420471191, 0.3339548408985138], [-0.9201843738555908, 0.44973671436309814, -1.854905128479004, -2.8221359252929688, 0.1023043841123581, -0.0008109070477075875, 1.1319693326950073, 0.5332962870597839, -0.9133643507957458, 2.8637638092041016, -1.3910996913909912, 0.97939133644104, 1.7149381637573242, 0.7473555207252502, 0.5737749338150024, 0.2950005829334259, -2.5958495140075684, -0.9839416742324829, -0.3277167081832886, 0.5262230038642883, 2.0002005100250244, -0.34697988629341125, 1.39763343334198, -1.759631872177124, -0.6444696187973022, -0.685600757598877, 1.1953206062316895, -0.659073531627655, -1.5946406126022339, -1.361611247062683, -0.16685649752616882, 0.4187372028827667, -1.1721972227096558, -0.2656407356262207, -0.6794115900993347, -0.3249249756336212, 1.1376137733459473, -1.7776405811309814, 1.2381528615951538, -0.27302786707878113, 0.4629806578159332, -1.8565694093704224, 0.6699018478393555, 0.7031117081642151, 0.4308432936668396, -1.4957716464996338, -1.684428334236145, 1.1523535251617432, -0.6594908237457275, 1.0073314905166626, -1.5371249914169312, 0.7636382579803467, 2.027419328689575, 0.005229055881500244, -1.9126014709472656, -0.003361861454322934, -0.7041181325912476, -0.6487642526626587, -1.9587398767471313, -0.33473101258277893, -0.5570545792579651, 0.6507397890090942, 0.32512468099594116, 0.3018707036972046], [1.1643110513687134, -0.6564472317695618, 0.014312547631561756, -0.16712023317813873, -1.0267165899276733, 0.3221510052680969, 0.8800310492515564, 0.8365903496742249, -1.4151651859283447, 0.44416722655296326, 0.2427779585123062, -0.34671005606651306, 0.012073948048055172, 0.3556034564971924, -0.577698826789856, -0.8318416476249695, -0.05290285870432854, -0.2643153667449951, -0.3917050063610077, -0.9056249856948853, -0.24771519005298615, -2.333956241607666, -0.44706183671951294, 1.0643116235733032, 0.7695497274398804, -0.024627910926938057, 1.4805036783218384, -0.46095985174179077, -0.30293893814086914, 0.6835089325904846, -1.9696111679077148, -0.9732828140258789, -1.1432151794433594, 0.9647780656814575, 0.4492082893848419, -1.1369342803955078, -0.5070006251335144, -0.760266125202179, 0.9752181768417358, -0.17377039790153503, 1.4178738594055176, 2.0178918838500977, 0.13912557065486908, 0.5003042221069336, -1.266550064086914, -2.0065066814422607, 0.004575113300234079, -0.07676324248313904, 0.32277342677116394, -0.6845635771751404, -1.0883513689041138, 0.1859421730041504, 0.9185464978218079, -0.05766218900680542, 0.09938043355941772, -0.882220983505249, 1.7439064979553223, 0.4852102994918823, -1.0221548080444336, -0.20179733633995056, 1.3355995416641235, -0.08382739871740341, -0.33532798290252686, 1.3534919023513794], [-0.38692009449005127, -0.9228776693344116, -0.6128681302070618, 1.9569096565246582, -0.08842018246650696, -1.1067798137664795, 0.6416773796081543, -3.622248411178589, -0.3085402250289917, -2.1964657306671143, 2.181772232055664, 0.21911846101284027, -0.26431334018707275, 1.1319029331207275, -0.7402514815330505, 0.32033172249794006, 1.4635684490203857, 0.18936021625995636, -0.6629041433334351, 0.1790749579668045, -2.2652347087860107, 0.27714449167251587, -0.14987488090991974, 0.9600579738616943, 1.060512661933899, -0.23262538015842438, 0.39386382699012756, 1.3661611080169678, 0.07012084871530533, -0.35323193669319153, 0.36602452397346497, 1.3845216035842896, -0.4619532823562622, 0.8722118139266968, -0.2289748191833496, 1.1747158765792847, -1.3301507234573364, -0.8333604335784912, -1.249840259552002, 0.2786390781402588, 0.9106736779212952, -1.3360029458999634, -0.11109858751296997, -0.3720628023147583, 2.6824142932891846, -0.4224953353404999, -1.0085850954055786, -0.01135215349495411, 0.49478086829185486, -0.6740438938140869, 1.0592656135559082, 0.8100461959838867, -0.5964751243591309, 0.1999269723892212, -0.2372588813304901, -0.24229943752288818, -1.5875134468078613, -0.3171027898788452, -0.5613448619842529, -0.005093245767056942, -2.2838141918182373, -0.7746151089668274, 0.19023564457893372, -1.2043259143829346], [-0.21117781102657318, -2.1166579723358154, 0.4442465603351593, 0.6019951701164246, 1.6435295343399048, -1.4124654531478882, 0.700127124786377, -1.5844061374664307, -0.32530075311660767, 0.11733917891979218, 0.006616014055907726, -0.2843754291534424, -0.49664127826690674, 0.4264385402202606, -0.8452927470207214, 0.016630394384264946, -1.4191144704818726, -1.1146047115325928, -0.1955324113368988, 4.045012950897217, 0.34008514881134033, 1.1895363330841064, -1.0797125101089478, 0.43796226382255554, -0.03840770199894905, 0.3758186101913452, 1.8355706930160522, 0.11310160160064697, -0.591134250164032, 2.611731767654419, -0.22839823365211487, 0.80757737159729, -1.205056071281433, 1.8390506505966187, 0.8988608121871948, -1.0905643701553345, -0.7153908014297485, -0.954149603843689, -0.870575487613678, 0.7734255790710449, 0.4493486285209656, -1.6690235137939453, -1.7581158876419067, 0.44547444581985474, -0.6862743496894836, 0.6790241003036499, -0.4888423681259155, -0.24944201111793518, 0.5249292254447937, -0.7252875566482544, 0.21945340931415558, 0.46907567977905273, 2.985151529312134, -0.4526517689228058, -0.6077710390090942, -1.2091280221939087, 2.0501351356506348, 0.333991140127182, -0.900929868221283, -0.9908259510993958, -1.9818748235702515, -1.014426589012146, 0.8305427432060242, 0.3977634310722351], [0.741089403629303, -1.281025767326355, -1.1620882749557495, 1.2025725841522217, -1.0168275833129883, 0.45990368723869324, 1.892719030380249, 0.17141467332839966, -0.4999188780784607, -0.5987020134925842, 0.6125032305717468, -3.780369758605957, -1.83296537399292, -0.11722645163536072, 0.7115054130554199, -0.12476003170013428, 0.9310857653617859, 1.1927924156188965, -0.11933797597885132, -1.6793123483657837, -0.43443089723587036, -0.5252108573913574, -0.39571085572242737, 0.8669679760932922, 2.1380529403686523, -0.6212000846862793, -0.24744835495948792, 0.19309143722057343, 0.042926911264657974, 2.0737509727478027, 0.8129636645317078, 0.7566689848899841, 0.1715501844882965, 0.021761996671557426, 0.5095231533050537, -0.5897373557090759, -0.887636661529541, 1.0313392877578735, 1.7148219347000122, -1.8177658319473267, -0.5512398481369019, 0.6435465216636658, -0.8672884106636047, -0.2587902247905731, -1.8871965408325195, 1.121985912322998, 0.7101526856422424, 0.21547625958919525, 0.27058660984039307, -0.33877843618392944, -0.50925213098526, 1.8279234170913696, -0.20129181444644928, -0.4430807828903198, 0.10129734873771667, 0.5268987417221069, -0.07559192180633545, -1.0347776412963867, -0.20608524978160858, 0.8490949273109436, 0.988646924495697, 0.10133960098028183, 1.7164149284362793, -0.32714247703552246], [0.9349385499954224, 0.1931019425392151, 1.0153194665908813, 2.1737613677978516, -0.6407312154769897, -1.9197394847869873, 0.35985058546066284, -0.6298087239265442, -0.42533501982688904, -0.17699608206748962, 1.424545407295227, 0.29307040572166443, 0.6349377632141113, -1.6995183229446411, 1.938862919807434, -1.6603065729141235, -0.458944708108902, 1.608883261680603, 0.5890485048294067, 0.6100780367851257, 2.593924045562744, 0.6422409415245056, 0.20343193411827087, -0.3830432891845703, -2.2076451778411865, 1.1029222011566162, 0.8841867446899414, -1.0850392580032349, -0.6746824979782104, 0.9018468260765076, 0.25808799266815186, 2.139634609222412, -0.1927584558725357, 0.6573877334594727, 1.4590367078781128, 1.0363991260528564, -2.2706894874572754, 0.12146667391061783, 0.4742712378501892, 1.0850516557693481, 0.2038169950246811, 0.07657463848590851, 0.25923964381217957, 1.3772574663162231, 1.0191766023635864, 0.8525121808052063, 0.2636323571205139, 2.354522943496704, 0.5979041457176208, -1.1776071786880493, 0.6219788193702698, -1.2598214149475098, -0.4328170418739319, 0.4698244631290436, 0.05118311196565628, 0.9128420948982239, 1.3095041513442993, -2.4229376316070557, 2.388364791870117, 1.4778351783752441, 1.8466356992721558, -0.3239016830921173, -0.6371859312057495, 0.8057687282562256], [-1.9062467813491821, 0.29970502853393555, -0.009476681239902973, -2.4651479721069336, -0.3290676772594452, -0.6226497888565063, 2.7725942134857178, -0.42220208048820496, 0.9438942074775696, -1.6656454801559448, 0.5361250638961792, -0.9333460927009583, 0.16931699216365814, -0.424221396446228, 1.1960868835449219, -0.4384889602661133, -0.10466565936803818, -0.027441468089818954, -1.1719880104064941, 1.7216523885726929, 0.3358372151851654, -0.20523148775100708, 1.509054183959961, -0.031593985855579376, -0.13477005064487457, -1.0470905303955078, 0.8697755336761475, 0.09543950855731964, -1.4875365495681763, 0.5617155432701111, 0.22376838326454163, -0.9969693422317505, 0.15807881951332092, 0.624151885509491, 0.33954301476478577, -0.5930396914482117, 1.0948759317398071, 1.2962956428527832, -0.7206969261169434, -1.5257666110992432, 0.10464632511138916, -0.5463775992393494, 0.011802068911492825, 0.5371401906013489, 0.06583549082279205, 0.5020423531532288, 0.7085879445075989, 0.7336816787719727, 0.4082498848438263, 0.3504064381122589, 0.4759634733200073, -0.6554578542709351, -0.8046567440032959, 0.3774396479129791, -1.8090085983276367, 0.08501705527305603, 0.0414762869477272, -0.7150893807411194, 0.13051216304302216, 2.453538179397583, -0.2003929764032364, -0.10883314162492752, 1.9863066673278809, 0.749190628528595], [1.0486501455307007, -0.704325795173645, 0.7059625387191772, -0.6458552479743958, 0.6375258564949036, 1.6232562065124512, 0.9588572978973389, 1.4405895471572876, -1.562129020690918, 2.0804431438446045, -1.1170395612716675, 0.8553887605667114, 0.5268974900245667, -1.3648780584335327, 0.9903318285942078, -0.42635372281074524, -1.3557579517364502, -0.026775004342198372, 0.6132316589355469, -0.6912532448768616, 0.2758570909500122, -1.0322372913360596, -2.397439956665039, 0.1042914167046547, -0.4547830820083618, -0.39434173703193665, -0.33904796838760376, 0.3737792670726776, 1.691348671913147, 0.9851545691490173, -0.07475830614566803, -1.4361501932144165, -0.5719310641288757, 0.06391323357820511, -1.8448139429092407, -0.5138535499572754, -0.6633569002151489, -0.5405872464179993, -0.8600982427597046, -0.7468162775039673, 0.024178599938750267, 0.2831522524356842, -0.6697967648506165, 0.029022328555583954, -0.3671667277812958, 0.5308204293251038, -0.46539610624313354, 0.2508130371570587, 1.0491379499435425, 0.040913261473178864, 0.7078914046287537, 0.08863557875156403, -0.5326821804046631, 0.2772330939769745, 1.2424163818359375, 0.929700493812561, -0.30998748540878296, -0.5472635626792908, -0.8388708233833313, -0.25006258487701416, 1.5874698162078857, 0.13306951522827148, -0.24225115776062012, 0.11358749866485596], [-0.4237864315509796, 1.3250048160552979, -0.5988628268241882, -0.5888978242874146, 1.2681968212127686, 0.8565046191215515, 0.3002113699913025, -0.14977921545505524, 0.22933141887187958, 0.2897564172744751, 0.22288860380649567, 1.1789909601211548, -0.3944668769836426, -0.8090301752090454, 0.07148079574108124, 1.4641356468200684, -0.16413158178329468, -0.5571735501289368, 1.1364940404891968, -0.019537096843123436, -1.117531418800354, 0.6185115575790405, -1.8851652145385742, 0.3653899133205414, 0.13098599016666412, 0.5524599552154541, -0.8668637871742249, 1.1988319158554077, -1.2635345458984375, 0.543814480304718, 0.16666382551193237, -0.19813914597034454, 0.8756235241889954, 1.1272650957107544, 1.5916086435317993, -1.5253890752792358, 2.704005002975464, 0.5712956190109253, -0.2072192281484604, -1.6929322481155396, 1.6517854928970337, -0.04835537075996399, 0.43154940009117126, -0.06478794664144516, 2.037383794784546, 0.6321626901626587, 1.1387542486190796, -0.30601733922958374, -1.3242226839065552, 0.5662978887557983, 1.1034057140350342, -1.5982681512832642, -0.7701392769813538, 0.14857624471187592, -1.7440526485443115, -0.06535562127828598, 2.233597755432129, -0.5278595685958862, -1.1508758068084717, 0.6717910766601562, 1.69212007522583, -1.4971730709075928, -0.23616327345371246, -1.3512635231018066], [0.4178647994995117, 0.3943730890750885, 0.6814548969268799, -0.030016353353857994, -0.6815356016159058, -0.051317229866981506, 0.8846616148948669, -0.533128023147583, 0.7833465933799744, 0.6913678050041199, 0.4654335379600525, 1.709262490272522, 0.1058279499411583, -0.7883574366569519, -0.8301486372947693, -0.1369703859090805, 0.38321027159690857, -0.03521677479147911, 0.7639405131340027, 0.4165252149105072, 0.7700212597846985, 0.5803742408752441, 0.18978552520275116, 0.31012362241744995, -0.8883673548698425, 0.7887790203094482, -0.9416744112968445, 1.5745857954025269, 0.9327065944671631, 1.3080897331237793, 1.0207812786102295, -0.07080704718828201, 0.7385315895080566, -0.003175100777298212, 0.33910396695137024, -0.4213920533657074, -1.3126356601715088, -0.4250030815601349, -0.6369978785514832, 0.7360244989395142, 0.908333957195282, 2.3106534481048584, -0.01650320366024971, 2.3219335079193115, -0.498654305934906, -0.2817046642303467, -0.9569088816642761, -0.2087508589029312, -3.082608461380005, 0.377288281917572, -0.3354222774505615, -0.1956293284893036, -0.8281452059745789, 1.281030535697937, -0.025669291615486145, 0.9883109331130981, -2.5369181632995605, -0.8993885517120361, -1.2854275703430176, 0.8740454316139221, -1.725242257118225, 0.23355475068092346, 0.3465588092803955, -0.04406224191188812], [0.2783532738685608, 0.26531749963760376, -1.2536256313323975, 0.18451932072639465, 2.1351921558380127, 1.3038508892059326, -0.9347109198570251, -1.2378966808319092, -0.6340157985687256, -0.6026020646095276, 0.5728614926338196, -0.13663674890995026, 1.1578518152236938, -0.7549383640289307, 1.2203021049499512, -1.1891539096832275, -0.5774011015892029, 0.3768343925476074, -0.3228304386138916, 0.4009157121181488, 0.25482499599456787, -1.9645648002624512, 1.6889362335205078, 0.9363954067230225, -0.8919880390167236, 1.1665986776351929, -0.9765253067016602, 0.2059992104768753, -0.6851589679718018, 0.748824417591095, 0.2789400517940521, 0.20004357397556305, -0.13535797595977783, 0.2578413188457489, 1.116955041885376, -0.6032783389091492, 1.6477323770523071, -0.13111084699630737, -0.35958608984947205, 0.9584341645240784, -0.49292024970054626, -2.0882813930511475, -0.42903903126716614, -0.09330139309167862, 0.6463246941566467, -1.216720461845398, -1.0303512811660767, 0.8067613244056702, -0.5931670665740967, 0.16217480599880219, -0.6559223532676697, -0.6450318098068237, -0.6210865378379822, 0.7944040894508362, -0.0937977209687233, -1.1544452905654907, -1.080580234527588, 1.715274453163147, -0.33620455861091614, -0.8188515305519104, 0.39995357394218445, -0.7458269000053406, -0.9399980306625366, 1.3589682579040527], [0.2978942096233368, 0.5237749814987183, -2.3798673152923584, -1.6226651668548584, 1.434234380722046, -0.8179619908332825, 0.7649399638175964, 0.3459760546684265, 1.5847116708755493, -0.2218790203332901, -1.8621610403060913, 1.3923113346099854, 1.8687520027160645, 0.7806904911994934, 1.3726983070373535, 0.7696159482002258, 0.38797637820243835, 0.8462126851081848, -0.7551984190940857, -0.781733512878418, 0.03476328402757645, 0.5100517868995667, -0.48047536611557007, 0.6784482002258301, -0.7208198308944702, 0.769886314868927, 0.16630148887634277, -0.5866801142692566, 0.12402530759572983, -0.9167512655258179, 0.4363308250904083, -0.9020577669143677, -0.6208164691925049, -1.3837254047393799, 0.07835646718740463, -0.941994309425354, 0.8885630369186401, -0.2009868323802948, 0.9531621336936951, 0.0005187953356653452, 0.724177360534668, 1.0864288806915283, -1.7211500406265259, -1.6106951236724854, -0.5737737417221069, 0.061894193291664124, 1.2011085748672485, -1.184104323387146, 0.591113805770874, 0.5114120841026306, 1.856945276260376, 0.06685705482959747, 0.4154938757419586, -0.43696051836013794, 0.7779344916343689, 1.1564120054244995, 0.4309852123260498, -1.2653430700302124, -0.9818424582481384, 1.2928247451782227, 1.5348060131072998, 0.06148091331124306, 1.6098695993423462, -1.3978121280670166], [-0.5098088383674622, 0.7477157115936279, 0.0035838792100548744, 1.3103368282318115, -0.391136109828949, 0.502869188785553, -2.2691423892974854, -0.989069402217865, -0.17352081835269928, -1.5156276226043701, 0.19796817004680634, 1.5121185779571533, 0.8899756073951721, 0.36667177081108093, -1.929932713508606, -1.1694116592407227, 0.04039071872830391, 1.223351001739502, 0.1405290812253952, -1.0324664115905762, 1.819562315940857, 0.2724096477031708, 1.2418491840362549, 0.9569610953330994, -0.8481897115707397, 0.7160223126411438, 0.7068348526954651, 0.247178316116333, 0.33145037293434143, 1.0539413690567017, 0.792137861251831, -0.07763835042715073, -1.6135767698287964, 0.31534984707832336, 0.26020941138267517, 0.8555554151535034, 0.465307354927063, 0.23018379509449005, -0.9781168699264526, 0.10063669085502625, -0.6566463112831116, -0.9679588675498962, 0.9258113503456116, -1.035789966583252, -1.236076831817627, 1.8288521766662598, -1.8129826784133911, -0.7882872223854065, -0.13517622649669647, 1.261336088180542, -1.02004873752594, -0.30275431275367737, 0.4621882438659668, -1.020026683807373, -1.3588236570358276, -1.2660409212112427, -0.0465143658220768, -0.5229670405387878, -0.9476861357688904, -0.6498282551765442, 0.9294466376304626, -0.5356264114379883, 1.1439448595046997, -1.3102695941925049], [-0.18615709245204926, 2.0249903202056885, -1.4970500469207764, -0.12655025720596313, -1.5908279418945312, 0.16160359978675842, -0.502907931804657, 0.24771246314048767, 0.402521550655365, 0.07837735861539841, 1.5250575542449951, 0.5116947889328003, 1.1455427408218384, 0.08662687242031097, 1.8157602548599243, 0.4794827401638031, 0.6382212042808533, -1.610599160194397, 0.14279265701770782, -0.859524667263031, -0.5949985980987549, -0.41898632049560547, -1.887900948524475, -0.5192300081253052, 0.13856559991836548, -0.7397438287734985, 0.45125630497932434, -0.6137722134590149, 0.06446267664432526, -0.5346193313598633, 0.5373079180717468, -0.1576690971851349, 1.3010443449020386, 0.5126755833625793, 0.18126161396503448, -0.7817521691322327, 1.5626353025436401, 0.8590770363807678, 0.9120253920555115, -0.3585752248764038, -1.7281473875045776, -1.502012014389038, 1.501442551612854, 1.330878734588623, -0.12385061383247375, 0.43704062700271606, 1.1161140203475952, 0.4324450194835663, 1.229794979095459, 2.9865431785583496, 0.0019818884320557117, 0.6298481225967407, -0.23136252164840698, -0.5455808639526367, 0.31444332003593445, 0.14764738082885742, -0.9498602151870728, 1.4979523420333862, -0.6206620931625366, -2.098839282989502, -0.8291962146759033, -0.0854160264134407, 1.7001349925994873, 1.6599910259246826], [-0.0528399832546711, -0.23042406141757965, -0.5622968077659607, 1.534201979637146, -0.054529495537281036, 0.5546229481697083, 1.3462320566177368, 0.21286791563034058, 0.30867117643356323, -0.292659729719162, 0.5933443903923035, -1.1703574657440186, 0.769668698310852, 0.8824074864387512, -1.055474877357483, 0.2829408645629883, -0.10434138774871826, 0.3658203184604645, -0.5864465832710266, 0.5305975675582886, 0.633910596370697, -0.8584697246551514, 0.1126493364572525, -0.23741070926189423, 0.041833218187093735, -1.2174036502838135, -0.5137079954147339, 0.1150064468383789, 0.17980529367923737, 0.2961544692516327, 1.5612114667892456, -0.19105122983455658, -2.698920965194702, 0.6736370325088501, 0.21583516895771027, -1.045837163925171, -0.1415916085243225, -0.6195551753044128, -2.260922431945801, 0.41952788829803467, 0.5741998553276062, -0.7242456674575806, -1.265139102935791, -0.08817701041698456, 0.9120050668716431, 0.10135465860366821, 1.3935260772705078, -0.45827481150627136, 1.108083724975586, 0.42211464047431946, 0.4433377683162689, -0.9150336980819702, 2.738328695297241, -1.1795743703842163, 0.2575419843196869, -0.12988272309303284, 0.012727741152048111, -0.483234167098999, 2.412574052810669, -0.28786545991897583, -2.342445135116577, -1.2117688655853271, 0.461166650056839, 0.14588531851768494], [-0.7447870969772339, -0.977594256401062, -0.29095789790153503, -7.389875827357173e-05, -0.49383676052093506, -1.426647424697876, -1.0276180505752563, 0.6257044076919556, 0.04473705217242241, 2.189882278442383, -0.22428244352340698, -0.5471507906913757, -0.3536902070045471, 0.17790073156356812, -0.33085331320762634, -0.4487283527851105, 0.12978315353393555, -0.7108986377716064, -1.1189298629760742, 0.6898869872093201, -1.0476300716400146, -0.3192653954029083, -1.4910471439361572, -2.225633144378662, -0.2897542119026184, -0.6992614269256592, 0.11787163466215134, 0.13103772699832916, -0.34955379366874695, -0.04375014454126358, -0.25109174847602844, 2.1541690826416016, 0.2951158285140991, -0.18775805830955505, 0.9910837411880493, -0.27232420444488525, 0.7756286263465881, 1.3755261898040771, -0.515299379825592, -0.49791786074638367, 0.35775992274284363, -0.14336222410202026, -0.29461532831192017, -0.1729632467031479, 0.26242780685424805, -0.2800140380859375, -1.485426425933838, -0.2672824561595917, -0.34507042169570923, 0.28085267543792725, 2.329857587814331, 0.16667845845222473, 0.26438456773757935, 1.1808161735534668, -0.817123532295227, 1.7284399271011353, 0.3472692370414734, 1.4861595630645752, -0.35862794518470764, -0.14516037702560425, 0.048304807394742966, -0.9882580637931824, -1.1340411901474, -0.6000235676765442], [-0.7692153453826904, -0.6063807010650635, 1.4763342142105103, -0.8810150623321533, 0.4467349946498871, 0.722884476184845, -0.29553812742233276, 0.84663987159729, 0.42441320419311523, 1.3164606094360352, -0.07051816582679749, -0.6271811127662659, -0.6204988360404968, -0.40498197078704834, 0.703429639339447, -0.42376428842544556, 1.3037916421890259, 0.1397377997636795, -1.8684275150299072, -0.2892453968524933, 0.8266562819480896, 0.271348237991333, -0.6191937327384949, -0.9162452220916748, -0.5563423037528992, -1.617571473121643, -0.4449978768825531, 0.16301031410694122, -0.6542853116989136, 0.7038705945014954, -1.27772057056427, 1.7238808870315552, 0.5771571397781372, -1.2761039733886719, 0.5257704854011536, 0.12974770367145538, 0.8689771890640259, 0.08120695501565933, 0.7088941931724548, 0.08072222024202347, 0.260181188583374, 0.8278999924659729, -0.42377397418022156, 0.37050244212150574, 1.0706638097763062, -1.2025470733642578, -0.5338753461837769, -1.1177940368652344, 0.6676778793334961, 0.41711539030075073, -0.43455034494400024, 1.2882989645004272, -0.8858525156974792, -0.5489370822906494, 0.11814533174037933, -0.3129550516605377, -0.9471689462661743, -1.968996524810791, -0.943309485912323, -0.5702407956123352, -1.1522810459136963, -0.05873006954789162, -1.4900414943695068, -0.09050337970256805], [-0.8042503595352173, -1.2945494651794434, 1.3202147483825684, -0.303164541721344, 0.5955912470817566, -1.678828239440918, -1.2509955167770386, -1.162424087524414, 1.4926738739013672, -1.5738743543624878, 0.46982342004776, 1.3848907947540283, -0.6228408217430115, -1.377235770225525, 0.9595008492469788, 0.5353087186813354, -0.5496262907981873, -0.392600953578949, 1.3522000312805176, 0.09120313078165054, 1.3476837873458862, -0.7799518704414368, 0.710935652256012, -0.37102940678596497, 0.8335235118865967, -1.0681092739105225, 1.000105857849121, 0.6931619048118591, 0.4242663085460663, -0.868068516254425, -0.2856813669204712, -0.4505404531955719, -1.0417120456695557, 0.3349919617176056, 0.6405808925628662, 1.3821479082107544, -0.09173881262540817, 0.6488135457038879, -0.12905775010585785, -0.8373635411262512, 0.24124698340892792, 0.37106573581695557, 0.1721188724040985, 0.7353982329368591, 0.2339513599872589, 2.1896188259124756, -0.7495639324188232, -0.6108752489089966, 1.5540143251419067, 0.5741794109344482, 0.050287455320358276, -0.2621978223323822, 0.3062704801559448, -0.20168034732341766, 0.4857006371021271, -0.08528590947389603, 0.4777834117412567, 0.2467518150806427, -1.188586950302124, 1.3811243772506714, -1.021547794342041, -0.4775063395500183, -0.8924959301948547, 1.0827423334121704], [-0.24871572852134705, 1.936601161956787, 0.1741592437028885, 0.7971805930137634, 0.31093287467956543, 0.7100008130073547, -1.3940225839614868, -0.1564556062221527, 0.25935015082359314, -0.3226301372051239, -0.38954803347587585, 0.24632175266742706, 0.5647870898246765, -1.0203068256378174, -0.9493695497512817, -1.2997201681137085, -1.0505330562591553, -1.9397010803222656, -0.3236064910888672, 0.5764560103416443, 0.7255765199661255, 0.3660486340522766, 0.013161963783204556, 0.9701854586601257, -0.25398582220077515, 0.38628432154655457, -1.26198410987854, 0.29555678367614746, 1.590824007987976, 0.6948754191398621, -0.3950917720794678, -2.2027370929718018, -0.7116580605506897, 1.7605867385864258, -0.178690567612648, -0.17237938940525055, -0.12833906710147858, -0.4269162118434906, 0.8098301291465759, 1.8348346948623657, -2.765211820602417, -1.8251674175262451, -0.087032251060009, -0.5073590278625488, -0.4302898943424225, -0.004043745808303356, -2.0186243057250977, -1.011289358139038, -0.10139065235853195, 0.5696330666542053, 0.8845511674880981, 0.6667729616165161, 1.4640628099441528, -1.910634994506836, 0.5651768445968628, -0.167599156498909, 0.912976861000061, 0.10884738713502884, -0.8735001087188721, 0.5257844924926758, -0.818470299243927, 0.7650599479675293, -0.5930074453353882, -1.5628098249435425], [1.7121723890304565, 0.5131447911262512, 1.686003565788269, -0.00743433041498065, -1.7878422737121582, -0.03095436468720436, 0.5710375308990479, 1.5752296447753906, -1.1642677783966064, -0.6294512748718262, 0.955277681350708, -1.0542548894882202, -0.7953141331672668, 1.9895151853561401, -0.2571645975112915, -0.6285897493362427, 1.5599972009658813, -2.7396206855773926, -1.1886695623397827, 1.1141750812530518, -0.027856387197971344, 0.01057798694819212, -1.2894097566604614, 0.44580477476119995, -0.537013590335846, -1.0688369274139404, 0.43108099699020386, -1.4113634824752808, 1.5614213943481445, 0.3582630753517151, 0.26548531651496887, -1.3588544130325317, 0.9033397436141968, 1.3638601303100586, 1.111961841583252, -0.8109053373336792, -0.15778382122516632, -0.668732762336731, -2.9617128372192383, -0.4321456253528595, 1.3430179357528687, 0.49215462803840637, -0.32822269201278687, 0.02703615091741085, -0.34731531143188477, 1.5773123502731323, -0.5799897313117981, -0.08859675377607346, 0.5038354992866516, 1.4297637939453125, 0.9527431726455688, 1.4265496730804443, 1.564595103263855, -0.2364683300256729, 0.024876190349459648, 0.7609905004501343, -0.44971486926078796, 0.080225870013237, 0.12703301012516022, -0.42017844319343567, -0.6054940223693848, -0.5948993563652039, -0.16403961181640625, -1.966135025024414], [1.0654973983764648, 0.3232012987136841, 1.7435494661331177, 0.7129783630371094, 0.10505491495132446, -0.6690895557403564, -0.9051884412765503, -0.49205395579338074, 1.5385568141937256, -0.40699103474617004, -0.13269822299480438, 0.4074292778968811, -1.147229552268982, 0.6562920808792114, -0.3536183536052704, -0.5828595161437988, -1.5072637796401978, -0.23806799948215485, -0.35672447085380554, -0.6518328785896301, 0.05188990384340286, 0.39878472685813904, -0.014738344587385654, 0.09809412807226181, -1.5233367681503296, 0.2993764281272888, 0.5219330191612244, 0.5360190272331238, -0.08974620699882507, 0.9394487142562866, -0.18636497855186462, -0.4592318534851074, 2.134984016418457, 1.3747605085372925, -0.8644351363182068, -2.275404214859009, 2.2721900939941406, 1.7433093786239624, -0.44867345690727234, -1.060208797454834, 0.201941579580307, -1.1466155052185059, 0.508612334728241, -0.3157082200050354, 0.15932781994342804, 1.1940630674362183, -0.6624647378921509, -0.07098745554685593, 1.6482967138290405, 0.8780861496925354, 0.20109789073467255, 0.8382270336151123, 0.9316517114639282, 0.41588926315307617, -0.462741881608963, 0.983591616153717, 0.9906148314476013, -1.8367544412612915, -0.21627767384052277, -1.2299370765686035, -0.1421232521533966, 0.16452781856060028, -1.208720326423645, 0.6285180449485779], [-0.26586776971817017, 1.4683908224105835, 0.5923452973365784, -0.9628514647483826, 0.6767902374267578, -1.5441333055496216, 0.10622542351484299, -0.10474901646375656, -2.1258835792541504, 0.10405807942152023, -2.163304328918457, 0.33968061208724976, 0.8632633090019226, -0.05634589120745659, -0.03390416502952576, -1.7586545944213867, -0.19535182416439056, 0.26879680156707764, 0.3906875550746918, 2.6328670978546143, -1.2772423028945923, 0.09839238226413727, -0.14191952347755432, 0.30003342032432556, -0.9912322163581848, -1.2490867376327515, 1.117484211921692, 0.48888900876045227, 0.603274405002594, -1.7739412784576416, 0.45061999559402466, 0.4690549671649933, 1.9936035871505737, -0.7068880200386047, -1.2635891437530518, -0.16340969502925873, 0.9723454713821411, -0.6115027070045471, -0.3785644471645355, 0.22790014743804932, -0.15547524392604828, 1.86830735206604, 0.36715126037597656, -0.8564863204956055, -0.6466583609580994, 1.5274511575698853, -0.0009158732136711478, -0.3114556074142456, 0.7345840930938721, 0.6796251535415649, -0.13878916203975677, -0.8009812235832214, -0.7883121967315674, -0.3731711506843567, 1.1107444763183594, 0.5071760416030884, -0.25385454297065735, 0.31856629252433777, -1.258162498474121, 0.13517256081104279, -1.1549814939498901, 0.7170272469520569, 0.6025448441505432, -0.6339030861854553], [-1.3528143167495728, -0.7493643164634705, -0.5977404713630676, -0.2485879808664322, 0.4666737914085388, -0.2885114252567291, 0.28367435932159424, 1.1025694608688354, 0.7382574081420898, -1.0462486743927002, 0.10976189374923706, -0.05444679036736488, -0.6383337378501892, -0.6944683194160461, 0.8623993992805481, -0.18976090848445892, -1.5750163793563843, 0.5687994956970215, -1.7848141193389893, -0.18148255348205566, 0.38284558057785034, 1.821031928062439, 0.1663619726896286, 2.2642791271209717, -1.3756392002105713, -0.08059458434581757, 0.23092994093894958, 0.7847293615341187, 0.045830436050891876, 0.07751408964395523, -0.6370860934257507, -0.9799069166183472, 0.4136795699596405, 0.8018761277198792, -0.004556416999548674, 0.09578593820333481, 0.5828792452812195, 0.9599471092224121, -0.17182502150535583, 1.2554290294647217, 0.22176945209503174, -1.4867944717407227, -0.871644139289856, 2.0003910064697266, -1.360770344734192, 0.7795959711074829, -2.115661382675171, 1.2135404348373413, -0.2603612542152405, 0.08538195490837097, -1.0322221517562866, -0.23803900182247162, 0.9357430934906006, 0.487505167722702, 0.6883608102798462, 0.4896266460418701, 0.579971969127655, -0.2524535059928894, -0.42419418692588806, -1.215574860572815, 1.408164620399475, -1.5170607566833496, -1.038305401802063, -0.2399805337190628], [1.0404460430145264, -0.8656365275382996, 0.10441449284553528, -0.5549328923225403, 0.44204169511795044, 0.8014360070228577, 0.9724938273429871, -1.6548210382461548, 1.971372127532959, 1.2954293489456177, -1.728584885597229, 0.47241973876953125, 1.1647096872329712, -1.0971800088882446, -1.1171622276306152, 1.4824103116989136, 0.5684694647789001, -0.896296501159668, -0.8300755620002747, 0.09932109713554382, -1.6070919036865234, -1.2367650270462036, 0.10204033553600311, -0.6291044354438782, 0.592206597328186, -0.16355814039707184, 0.017907297238707542, -1.6995794773101807, 1.5028027296066284, -1.4839941263198853, 0.9996225833892822, 0.6322068572044373, 0.10386193543672562, 1.629357933998108, 0.8662735223770142, -0.5098857879638672, -2.1573007106781006, -0.7017432451248169, -0.39575833082199097, -0.624883770942688, 0.041901979595422745, -1.45220148563385, -0.6437443494796753, -1.2759639024734497, 0.8653854131698608, -1.564424991607666, 1.5176689624786377, -0.8141146302223206, -1.0331956148147583, -1.487815499305725, -0.1272018700838089, -0.689602792263031, 0.26364609599113464, -0.7976334095001221, -0.11872421205043793, 1.83246910572052, 0.7994223833084106, 0.32137855887413025, 1.6532940864562988, -0.3654027283191681, 0.40605539083480835, -1.081803321838379, -1.1755893230438232, 0.6425695419311523], [2.572734832763672, -1.0456880331039429, -0.2993640601634979, -1.4017612934112549, 1.1615943908691406, 0.18393675982952118, -0.40920746326446533, -0.24406352639198303, 0.12670515477657318, 2.5952250957489014, -0.641055166721344, -1.0871820449829102, -0.9783411622047424, -0.9361016154289246, -2.516845226287842, -0.18481223285198212, -1.1283050775527954, 1.2821801900863647, 0.5049265623092651, -0.7448603510856628, 0.37139400839805603, -0.8575795888900757, -1.0105857849121094, 0.17430098354816437, -1.4013258218765259, -0.09788037091493607, -0.528684139251709, -0.6804410219192505, -0.6555832028388977, 0.09795284271240234, 0.40386903285980225, 0.7158191204071045, -0.7446654438972473, -0.654026985168457, 0.13591459393501282, 1.0705606937408447, 1.5447994470596313, -0.015472885221242905, -0.5706194043159485, -1.2989394664764404, -0.37662267684936523, -0.5424926280975342, -0.31051915884017944, -0.045952390879392624, 0.6075478792190552, 0.7323659658432007, -1.2767627239227295, -1.0019358396530151, -0.43469515442848206, -0.8820720314979553, -0.5193251967430115, 0.9703376889228821, -0.007117168512195349, -1.2391448020935059, 0.6404967308044434, -0.18310856819152832, -0.5637940764427185, -0.05964917689561844, 0.5429302453994751, -0.8064906001091003, 0.6697887182235718, -0.47160542011260986, 1.3975539207458496, 1.5641001462936401], [0.9429777264595032, -0.08028052002191544, -0.44121354818344116, -1.1700364351272583, 0.0597275048494339, 0.5445529222488403, 0.3876461684703827, 0.3408842980861664, -0.1131475493311882, 1.327678918838501, -0.2816081941127777, 0.22988422214984894, 0.9763239026069641, 1.1082432270050049, -1.1365464925765991, 0.6090235710144043, 0.3753822147846222, 2.364793062210083, -0.999206006526947, 2.4501616954803467, -0.3711131811141968, 1.683505892753601, 0.5736514329910278, 0.7581833600997925, 1.3766354322433472, -2.4229774475097656, 0.23505541682243347, 0.5617250800132751, -0.5385450124740601, -0.5831053853034973, 1.837785005569458, -0.2588745057582855, 1.6108050346374512, 0.23176060616970062, -0.5899096131324768, 0.33536627888679504, -1.3388471603393555, 0.19904187321662903, 0.5952904224395752, 0.17728310823440552, -0.8062803745269775, -0.127206951379776, 1.543399453163147, -0.6333659291267395, -0.1822396218776703, 0.6110872626304626, 1.173352599143982, 1.436513900756836, 0.705272376537323, 0.790123701095581, 0.37233591079711914, -0.10545074939727783, -0.7598757147789001, 1.230913758277893, -0.07925291359424591, -0.9352471828460693, 1.4867652654647827, -1.3043498992919922, -1.1939105987548828, 0.9075880646705627, 0.14791031181812286, 0.06539633125066757, -0.739865243434906, 0.751310408115387], [-1.1774653196334839, -0.12983489036560059, 0.39769959449768066, -1.2776063680648804, 0.44806602597236633, 2.3645293712615967, 0.3890867531299591, 0.025324726477265358, -0.2281070351600647, 0.3270869553089142, -0.07004545629024506, 0.14264802634716034, 0.2013871669769287, 0.4117608964443207, 0.08988255262374878, -1.6714128255844116, 0.06050414219498634, 0.523661732673645, 0.5641236901283264, 2.499934434890747, 0.2601413428783417, 0.7576013803482056, -2.1349198818206787, -0.25933074951171875, 0.3380086421966553, 0.8354272842407227, -1.5596565008163452, 1.4494988918304443, -1.5404974222183228, 0.13988074660301208, 2.2433125972747803, 0.2542519271373749, 0.3224227726459503, -0.06595085561275482, 1.6890043020248413, -0.7440881729125977, 0.2222333401441574, 0.036409247666597366, 1.8549857139587402, 0.2800826132297516, 0.35173293948173523, 0.748185932636261, -0.10292982310056686, 0.6015137434005737, 1.036879062652588, 0.7995977997779846, -0.05089472979307175, -0.21543386578559875, -0.9429036974906921, 0.8948267102241516, 1.7119814157485962, 0.05294332653284073, -0.13606153428554535, -0.2820596396923065, -0.343425452709198, 0.005005960818380117, -0.7012264132499695, 0.6032384037971497, -1.0610755681991577, 2.4510321617126465, 1.1907151937484741, -0.43487635254859924, -0.5150336623191833, -0.8595637679100037], [-2.0750725269317627, 0.3995007872581482, -0.7170001268386841, 1.0021417140960693, -1.2052186727523804, 2.180340528488159, -1.3376035690307617, -1.1790692806243896, 0.801680326461792, 0.7429314851760864, -0.11058961600065231, -0.7200960516929626, -1.7450268268585205, -0.3480098843574524, -2.1887736320495605, -1.9998477697372437, 0.622681736946106, -1.5677459239959717, -0.6613674759864807, 0.3103434145450592, -1.473957896232605, 0.8714151382446289, 0.10567545890808105, 0.07122338563203812, -0.28036272525787354, -0.2245568484067917, -1.090986967086792, -0.9832661747932434, 0.3476226031780243, -0.7602332830429077, -0.3184002637863159, -0.36797043681144714, 0.54205322265625, -1.7800674438476562, -0.03460179269313812, -2.0356621742248535, 1.3477739095687866, -0.5418639779090881, 1.0197274684906006, 0.5910235047340393, 0.48254483938217163, -1.0700781345367432, 0.028474600985646248, -0.22822202742099762, -0.6615813374519348, 0.4418541193008423, 0.42724406719207764, 0.34728625416755676, -0.9041464924812317, 0.12392793595790863, 0.42635127902030945, 0.3605954647064209, 2.3533408641815186, 0.9309248328208923, 0.6160346865653992, 0.23188404738903046, 0.4817817211151123, -0.2942518889904022, -1.0935298204421997, -1.2663449048995972, 0.3506002724170685, -2.4237327575683594, -0.4393197298049927, -1.5347787141799927], [-0.5519554615020752, 0.598529040813446, -1.3725272417068481, -1.2995800971984863, -0.40215474367141724, -0.46981295943260193, 0.30462193489074707, -0.7735077738761902, -0.19900763034820557, 0.6156139373779297, 2.066786050796509, 1.1202529668807983, -0.39631736278533936, 0.394337922334671, 0.8427961468696594, -0.9689812660217285, 2.9035587310791016, 1.0831525325775146, -1.4427242279052734, -0.48099285364151, 1.3185983896255493, 0.5331097841262817, 0.6126037240028381, 0.6705796718597412, -1.169844150543213, -0.2826017737388611, 0.5231525897979736, -0.639089822769165, -1.6671396493911743, 0.3497997224330902, 1.7354369163513184, 0.059079982340335846, 0.3402956426143646, -0.8717041015625, -0.6393532752990723, -1.2879726886749268, -1.0772994756698608, 0.336911141872406, 0.9907758235931396, -0.5544384717941284, -0.13329868018627167, -1.5837366580963135, 0.5172232389450073, -0.2681652903556824, 0.7166070938110352, 0.9228309392929077, 2.168911933898926, 0.7293527722358704, -0.8473854064941406, -1.4606584310531616, -0.8819220662117004, -0.09356141835451126, 0.28319409489631653, -1.0115762948989868, 0.9046933650970459, -1.5303230285644531, -0.3264540433883667, -0.8779492378234863, 0.529157817363739, 0.18378080427646637, 0.44862279295921326, -0.014911804348230362, -0.1923133283853531, -0.93849778175354], [0.4673321545124054, -1.1714214086532593, -0.59052574634552, 1.7931122779846191, -0.9464338421821594, -0.1441459059715271, -0.12927484512329102, 1.1296193599700928, 0.7988805174827576, 1.3792037963867188, 0.057539790868759155, -0.603938639163971, -0.18121212720870972, -0.12243414670228958, 0.6269888281822205, 1.2580891847610474, -0.9957261085510254, 1.2714604139328003, -2.375196695327759, -0.9911881685256958, -0.24261920154094696, -0.03965495526790619, -1.3302077054977417, -0.820615828037262, 0.14221826195716858, 1.212934970855713, -0.23188498616218567, -0.27596327662467957, -0.08903771638870239, 0.870328426361084, -1.0803308486938477, -0.31449753046035767, -0.6429356336593628, -0.6009010076522827, -0.6185247302055359, 0.06365472823381424, 0.2893296182155609, -0.3966968357563019, 0.4257253110408783, 1.0636574029922485, 0.48747721314430237, 0.7612379789352417, 1.8880549669265747, 0.33194398880004883, 0.5903356671333313, -0.4883747100830078, -1.67127525806427, -2.321885347366333, 0.7343978881835938, -0.8137470483779907, -0.9520020484924316, -0.9949724078178406, -0.8852425813674927, 0.2425403594970703, -1.7669038772583008, 1.2260305881500244, -0.07136553525924683, 0.9310280680656433, -0.3134298026561737, -0.1692734658718109, -0.21921879053115845, 0.19444838166236877, 0.05570411682128906, -0.909756600856781], [-1.4966634511947632, 2.4209084510803223, -0.5959372520446777, 0.7506847381591797, -0.05910620838403702, 0.11082340031862259, 1.8098986148834229, -1.557934045791626, 0.2368791401386261, -0.04443742334842682, 0.24906322360038757, -0.086502306163311, 0.4438796639442444, 1.2910020351409912, 0.31688153743743896, 1.384552001953125, -0.3501259684562683, 2.154745101928711, 1.0069481134414673, 0.12144719809293747, -0.01194678619503975, 1.4549906253814697, -0.023657698184251785, -0.356023907661438, 0.5957074761390686, -1.1954381465911865, -1.2447553873062134, -0.9433251023292542, -1.217696189880371, 1.0156002044677734, -1.264060616493225, -0.15811887383460999, 1.4133285284042358, -0.5495219826698303, -0.4660460948944092, 0.3415762782096863, 0.21282696723937988, 0.717027485370636, 1.3256760835647583, -0.5952225923538208, -0.04756232723593712, -0.5422282814979553, 0.4911669194698334, 0.2709622383117676, 0.5530449151992798, -1.9187159538269043, -0.2405817210674286, 0.7959522604942322, -0.6983667612075806, 0.17887145280838013, -0.3321380913257599, -0.8270268440246582, 1.0633772611618042, -1.8668133020401, -0.26268520951271057, -0.9412689805030823, 0.8974816799163818, 0.8938801288604736, 0.8631002902984619, 0.4463561773300171, -1.111416220664978, -0.32966017723083496, -0.9659010767936707, -0.19941388070583344], [0.3549988567829132, -0.4409765303134918, -0.16442181169986725, -1.0384161472320557, -1.6283018589019775, -1.9883360862731934, 1.9690990447998047, 0.47520676255226135, 0.2029070258140564, 0.650575578212738, -0.5948252081871033, 0.7106277942657471, 0.5337258577346802, 0.9832951426506042, -0.6353533267974854, 1.6218633651733398, -0.7303594946861267, 0.5234587788581848, -0.46288779377937317, 0.021296171471476555, 0.1919013410806656, -0.9167904853820801, 1.4905000925064087, -1.5644738674163818, 0.22178012132644653, 0.7131661176681519, 2.9824252128601074, -0.9820484519004822, 0.2496415376663208, 0.2703971266746521, -1.387282371520996, 1.0366429090499878, 0.3283007740974426, 0.35795286297798157, 1.3071269989013672, -0.39684391021728516, -0.3979214131832123, -0.03721960261464119, 0.6222699880599976, -0.07663459330797195, -0.7577324509620667, -1.1083821058273315, -0.7577719688415527, -1.0460596084594727, -0.29181385040283203, 0.21459995210170746, -0.8008707761764526, 0.32758232951164246, -0.9154449701309204, 0.9444926381111145, -0.09128042310476303, 1.1322212219238281, -1.2968685626983643, -1.4644454717636108, -0.22816695272922516, 1.4201014041900635, -1.275987982749939, -1.8286632299423218, 0.21619980037212372, -0.20376937091350555, 0.9543279409408569, 0.28988882899284363, -0.20855078101158142, -0.0787232518196106], [1.6367404460906982, 1.2328578233718872, 0.4130145311355591, 0.5998229384422302, 0.5405272245407104, 0.9963114261627197, -0.8259275555610657, -0.8771454691886902, -0.2022961527109146, -0.9000042676925659, -1.1749770641326904, 0.5731839537620544, 0.31879809498786926, 0.16815988719463348, -1.7067848443984985, -0.4461480379104614, -0.11972645670175552, -1.6888047456741333, 0.6545789241790771, 0.0806514173746109, -1.2254632711410522, -0.3836882710456848, -1.482772946357727, -0.9572225213050842, -0.45345500111579895, 0.8026559948921204, -0.7122413516044617, -0.9100554585456848, 1.780226469039917, -0.6389992833137512, -0.11389452964067459, 0.5642752051353455, -1.3707075119018555, 0.14561660587787628, 1.8260903358459473, 0.9873188138008118, -0.3756069540977478, 0.7144681215286255, -0.5197240710258484, -1.730342149734497, 1.2347052097320557, 0.9240363240242004, -1.087557315826416, 0.7667788863182068, -1.1842601299285889, 1.0309404134750366, 0.5674695372581482, -1.5535516738891602, 1.6668490171432495, -0.13112801313400269, 0.7064875960350037, -0.8544983863830566, 0.4057736396789551, 0.5322127938270569, -0.9940710067749023, -0.2633962333202362, 1.0982893705368042, 3.504230260848999, -1.736245036125183, -1.9186984300613403, 0.7124500274658203, -0.09460815787315369, -0.03816605731844902, 0.15354877710342407], [1.2453256845474243, -0.9019755125045776, -1.0084677934646606, -0.09573996812105179, -0.572553813457489, -1.2821617126464844, 1.745858907699585, -0.5055549740791321, 0.1888539046049118, -0.28337544202804565, 1.76939058303833, -0.5888053774833679, -0.8324096202850342, 0.2571237087249756, -0.901568591594696, -0.3977462351322174, 0.44674167037010193, -0.7037840485572815, -0.35842224955558777, -0.6782299876213074, 0.5348123908042908, 0.8667122721672058, -0.23765188455581665, -0.39588597416877747, -0.7381715178489685, -0.8341981768608093, -1.1273951530456543, -2.0363240242004395, -1.8223942518234253, -0.31992658972740173, -0.11654356867074966, 0.9550966620445251, -0.45582300424575806, 0.629260241985321, -1.0525670051574707, 0.7438746690750122, 0.4400458037853241, 1.2086777687072754, -0.4975060522556305, -0.9172511696815491, -0.664313793182373, 1.9949971437454224, 1.8258782625198364, -0.08122050762176514, 0.11245457082986832, -0.06191156432032585, 1.0236555337905884, -0.556994616985321, -0.4472934901714325, -0.8816508054733276, -0.26038873195648193, 0.387713760137558, 1.4667835235595703, 1.1593434810638428, 1.2894508838653564, 0.4297371208667755, 0.5908246636390686, -1.4975039958953857, 0.2135086953639984, -0.0632283091545105, 0.7557232975959778, -0.4614434242248535, 1.7537850141525269, 0.9033330082893372], [0.6895310878753662, -0.586880087852478, 0.8902010917663574, -1.2297818660736084, 0.6932713389396667, -1.6434334516525269, 0.5189444422721863, -0.6979364156723022, -0.7022705078125, -1.8750264644622803, -1.0280683040618896, -1.1373121738433838, -1.6665831804275513, 0.3885912597179413, 0.989942729473114, 0.6817575097084045, -0.2274511307477951, -0.5637586712837219, -0.4676932692527771, 0.4381087124347687, -0.007623604964464903, 0.9026121497154236, 0.48282286524772644, -0.4371722340583801, 0.7734593749046326, -0.5464771389961243, -0.42176446318626404, -0.9421760439872742, 0.7610149383544922, -0.26708880066871643, -0.8995826244354248, 1.1761170625686646, 0.9552769064903259, 0.8169974684715271, 0.7366641163825989, 0.04203368350863457, 0.9203495979309082, 0.4580586552619934, -0.3509315550327301, -0.9156836271286011, -2.0140039920806885, 0.6937548518180847, -0.6538196802139282, 0.12668339908123016, 0.7238518595695496, 0.8353814482688904, -1.3563112020492554, 0.3527049422264099, 0.6283425688743591, -0.09698030352592468, 0.076827771961689, -0.05163457617163658, -0.2525818943977356, -0.5083561539649963, 0.7092061042785645, 0.5039199590682983, -0.02933027222752571, 0.020215937867760658, 1.54127836227417, 0.25228312611579895, 0.2797565162181854, 0.723948061466217, -0.4625140130519867, -0.7705837488174438], [-0.6830839514732361, -1.5623551607131958, -0.7213125824928284, -0.383728563785553, -1.5316894054412842, -0.5012778639793396, 0.3427531123161316, -1.081383228302002, 1.4856595993041992, 2.0671277046203613, 0.5776785612106323, -0.36422985792160034, 1.5010743141174316, 0.1682843565940857, -0.4183304011821747, 0.5708932280540466, 0.7292349338531494, 0.26101627945899963, -0.42097362875938416, -1.3126130104064941, 0.22964878380298615, 0.11380138248205185, -0.5876508355140686, 0.5587761402130127, 0.6550445556640625, 0.20543590188026428, 0.5612763166427612, -0.8510451912879944, 0.1920587122440338, 0.37438300251960754, -0.32828956842422485, 0.5410509705543518, -1.2176891565322876, 0.5095018744468689, 1.4378631114959717, -0.40959432721138, -0.5472493171691895, 0.17734937369823456, -0.564233124256134, 0.0015731911407783628, -0.6799747943878174, 0.6164829134941101, -0.9895339608192444, 0.13656172156333923, 0.5149427056312561, 1.3975986242294312, 0.6699212789535522, -0.43451130390167236, 0.5539435148239136, -0.30867311358451843, 0.3000783622264862, -0.19500420987606049, -1.2143278121948242, -0.7395802736282349, 0.02888147346675396, 0.3773666322231293, -1.1051361560821533, -0.49935460090637207, -0.6349321603775024, -0.9612287878990173, -0.671699583530426, -0.9353342652320862, -0.841802179813385, 1.7094433307647705], [-0.3673327565193176, 0.4624900221824646, 1.0463404655456543, 0.7881125807762146, -0.9850111603736877, -1.2376139163970947, -1.312002420425415, -1.0787237882614136, -0.806569516658783, -0.6156293749809265, -0.11064350605010986, -1.4726536273956299, 0.28781893849372864, 0.07338854670524597, 0.8148302435874939, 0.5371968746185303, -0.6071001887321472, 0.37833017110824585, 2.0167243480682373, 2.4778637886047363, -0.03487464413046837, 1.0002590417861938, 0.0732605904340744, -0.2213544249534607, -0.3325830399990082, 0.5991353988647461, -0.9274322986602783, -1.5643997192382812, -0.7231876850128174, -0.6792182922363281, -0.9040118455886841, -0.05356479436159134, -0.7469427585601807, -1.494834303855896, -1.9739375114440918, 1.3160558938980103, 1.0853062868118286, -2.1717076301574707, -1.2709461450576782, 0.7288288474082947, -1.9862923622131348, 0.35497787594795227, 1.4630581140518188, 0.34336018562316895, 0.4869476854801178, 0.8354756832122803, -0.08385052531957626, -0.536440372467041, -0.03619539365172386, -0.7879515886306763, 0.5026529431343079, 0.4326651096343994, 0.03885869309306145, -0.21199727058410645, -1.7916866540908813, 0.8608320951461792, -0.6242003440856934, 1.4139829874038696, 0.9301605820655823, -1.2530189752578735, -0.4618574380874634, 0.22417768836021423, 0.10746049880981445, -0.24840401113033295], [0.4310709834098816, -0.2545265257358551, -1.4478821754455566, -0.15209032595157623, 0.2479328066110611, -0.9985434412956238, -0.820547878742218, -0.4027642011642456, 1.7655084133148193, -1.567761778831482, -0.7354853749275208, -1.645557165145874, 0.11809515208005905, 1.3735040426254272, 1.195428490638733, -0.8755791187286377, 2.539750576019287, 0.9110323786735535, -0.9299696683883667, 0.5698387622833252, -0.9381320476531982, 0.4454107880592346, -0.9192086458206177, 0.814376175403595, 0.4465792179107666, -1.041934609413147, -0.5665107369422913, -0.4313527047634125, -1.0113557577133179, 0.09341409057378769, 1.375090479850769, 1.322364091873169, -0.9882069230079651, 2.5922281742095947, -0.9264849424362183, 0.6505000591278076, -0.7201734781265259, -2.0077579021453857, -1.230029582977295, -0.1063835620880127, -0.790918231010437, -0.07337742298841476, -0.1623959243297577, 0.03910541906952858, -0.6113994121551514, -0.29371365904808044, -0.455375999212265, 0.8691461682319641, 1.8088886737823486, 0.22179245948791504, 0.49965137243270874, 1.4887983798980713, -0.4144369065761566, 0.2967251241207123, 2.458366870880127, -1.976028561592102, -1.9346449375152588, -0.08766680955886841, -0.954802393913269, 0.20211799442768097, -0.17077623307704926, 1.5014222860336304, -2.3704254627227783, -0.45222434401512146], [1.084663987159729, 0.695509672164917, 0.2843233346939087, -0.4365997314453125, 0.3315924406051636, 1.0975390672683716, -0.0022192581091076136, -0.30448436737060547, -0.5779497027397156, 2.461503744125366, 0.11555499583482742, -0.3423399031162262, -0.6054108142852783, -1.2928354740142822, 0.5063169002532959, -2.238375186920166, -1.177549958229065, 1.0430970191955566, -1.0133090019226074, 0.7563396096229553, -0.26133739948272705, -1.0673775672912598, 0.24692904949188232, -1.6071127653121948, -0.4788077473640442, -1.0024431943893433, 1.309144139289856, -0.5483095645904541, -0.682405412197113, -0.4914814829826355, -0.17030923068523407, 0.40325024724006653, 0.2042153775691986, 0.6299431324005127, 1.3472059965133667, 0.8615168333053589, -1.1589581966400146, -0.6336285471916199, -0.7918089628219604, 0.3005744218826294, 0.8191276788711548, 0.04574495553970337, 3.3257992267608643, 2.6910626888275146, -1.7448309659957886, -0.5260188579559326, 0.3992462754249573, -0.2999085485935211, 0.19867932796478271, -0.8053200840950012, -0.5158354640007019, 0.5869922041893005, 1.7159050703048706, 0.44898897409439087, -0.5475728511810303, -1.2557055950164795, -1.8615714311599731, -0.4596366584300995, 0.9085851311683655, -1.6064037084579468, -0.016230789944529533, 0.2557111084461212, -1.818002700805664, 0.779323399066925], [1.4133487939834595, -0.5617157220840454, -1.807461142539978, 0.0681694895029068, 1.8629213571548462, 0.22987699508666992, -0.25888749957084656, 2.237919330596924, 1.5978797674179077, -0.9897364974021912, -1.1873313188552856, -0.29497388005256653, 0.36755967140197754, 0.5113284587860107, 0.6571378707885742, 0.21386724710464478, 0.9686682820320129, -1.442622184753418, 1.8999533653259277, 0.9874542951583862, -0.952400803565979, -0.3399767577648163, -0.9937711954116821, 0.21431377530097961, -1.0157320499420166, 0.7832773923873901, -0.056774914264678955, -1.742933988571167, 1.0798448324203491, -0.18403486907482147, -2.181276321411133, -1.1606435775756836, 1.3487012386322021, 1.1462205648422241, -1.5070048570632935, 0.33853963017463684, -2.415433168411255, -0.7220742702484131, 0.6605973839759827, 0.40339693427085876, -0.07464085519313812, -0.33833828568458557, -0.8773925304412842, 0.8967390656471252, 1.3230434656143188, -2.029421806335449, -0.05531973019242287, -0.31649208068847656, -0.7963689565658569, 1.0046113729476929, 0.6160570383071899, -0.7617776393890381, -1.4213685989379883, -0.8685987591743469, -1.2097501754760742, -0.7203015685081482, -0.6824731826782227, -0.34844425320625305, -0.7598379254341125, -1.510898232460022, 0.30569207668304443, -2.009080648422241, -1.4414242506027222, 0.12952883541584015], [1.8703964948654175, -2.3834779262542725, -1.185296893119812, 0.5500019192695618, 2.019094944000244, -0.1932092159986496, 0.9305261969566345, 0.601294994354248, -1.371801733970642, 0.7035632133483887, 0.8394228219985962, 0.8472376465797424, -0.6314070820808411, 1.6161842346191406, 2.038776159286499, 1.1407843828201294, -2.1093456745147705, 1.4931749105453491, 0.5465008616447449, -1.6595289707183838, 1.150801420211792, 0.1117861345410347, 1.7797924280166626, 0.5511265993118286, -0.5559041500091553, -0.7388608455657959, -0.41346171498298645, -0.0014721449697390199, -0.658456027507782, -0.010342409834265709, 0.2685801386833191, -0.6681356430053711, 0.44918808341026306, 0.12478349357843399, -1.1478831768035889, 0.49384650588035583, 0.09588979184627533, 0.7887367606163025, -0.6064233183860779, 0.6123205423355103, 1.5424202680587769, 0.5113853216171265, 0.8368658423423767, -0.8496375679969788, -1.835606336593628, 0.9828752875328064, -0.5663706660270691, 1.1428390741348267, 0.04446898400783539, -0.7060628533363342, -0.7421073317527771, 0.8716341853141785, -2.257629632949829, -0.61942458152771, -0.0922546237707138, -0.08613641560077667, -1.1536518335342407, 0.14943675696849823, -0.9149187207221985, 3.4545445442199707, 0.5812241435050964, -0.0616932213306427, 0.06168685108423233, -0.06696882098913193], [-0.37363529205322266, -0.49180302023887634, -0.6895829439163208, 0.18128907680511475, 0.5370656251907349, -0.643192708492279, 0.7957689762115479, 0.44184964895248413, -0.5616365671157837, 0.6138176321983337, 0.4786006212234497, -0.24612654745578766, 0.06453189998865128, -0.3879934549331665, 0.8369778990745544, -0.5324832797050476, -0.7558857202529907, -0.1309269219636917, 0.8643960952758789, -1.0826607942581177, 0.5250354409217834, 1.0976333618164062, 0.7378783822059631, -0.43463122844696045, 0.1678793579339981, -0.27506113052368164, 0.04881587252020836, -0.7026790976524353, 1.4859907627105713, 0.1888449341058731, -0.9716248512268066, 0.20916372537612915, 1.6120063066482544, -0.23763637244701385, -0.7132354974746704, -1.1136482954025269, -0.12642312049865723, -0.4929872751235962, 0.03772641718387604, 0.3997272849082947, 0.9126234650611877, -1.731566309928894, 0.13937750458717346, -0.34835392236709595, -2.4698326587677, -0.7868855595588684, 0.8719667792320251, 0.9349273443222046, -1.9100350141525269, 0.3576889634132385, -0.010925421491265297, -0.8399850726127625, 2.390984296798706, -0.9891757965087891, -0.05508618429303169, 0.11463320255279541, 0.16584302484989166, -0.9331324696540833, -0.34909185767173767, -1.4299874305725098, 0.22662563621997833, -0.8838135004043579, 1.7777286767959595, 0.4504319429397583], [0.9451244473457336, -0.6157305836677551, 0.2068629413843155, 0.6596314907073975, 1.048548698425293, -0.20112375915050507, -0.03417142108082771, -0.9032580256462097, 3.6241304874420166, -1.3191319704055786, -0.11811951547861099, 0.16382327675819397, -0.6119444966316223, -1.084097981452942, 1.816314697265625, 1.7429726123809814, -1.507521629333496, 0.765567421913147, 1.9605516195297241, -1.1661723852157593, -0.6960585117340088, 2.538217782974243, 0.6258023381233215, 0.753690242767334, 0.25787481665611267, -0.04938260838389397, -1.6454418897628784, -0.3739834427833557, 1.3163563013076782, 0.1587171107530594, 0.05856999382376671, -0.3408014178276062, 0.054000694304704666, 0.14900082349777222, -1.6050509214401245, -0.2737780213356018, -0.8954140543937683, -0.48545530438423157, 0.4077400267124176, 0.003954071085900068, -1.6006722450256348, -0.2267780900001526, -0.05777253583073616, -1.594667673110962, 3.195213794708252, 0.37477004528045654, 0.4725155234336853, -0.25911515951156616, -0.09523233771324158, 0.3071083128452301, 0.5217107534408569, 0.5020015835762024, 0.15506955981254578, -0.7509629726409912, 0.13184107840061188, -1.1335058212280273, -1.1525397300720215, 2.6059772968292236, -0.5193581581115723, 1.5390722751617432, 0.2958771586418152, 0.1305689811706543, -0.985675573348999, 0.4065307378768921], [-1.0422261953353882, -0.7938570976257324, 1.2683407068252563, 1.932192087173462, 1.5900832414627075, 1.0663036108016968, -0.07252668589353561, -0.40519973635673523, -0.8047012686729431, 0.5937728881835938, -1.2407833337783813, 1.1305981874465942, -0.6783642768859863, -0.12483986467123032, -0.25250953435897827, 1.1046874523162842, -0.35372185707092285, -1.4550983905792236, 1.6090327501296997, 0.3538707494735718, -0.018520496785640717, -1.2610524892807007, 0.4914555549621582, -0.41409826278686523, 0.8716464638710022, -0.10976877808570862, 0.6656259298324585, 2.098634719848633, -1.5010807514190674, -0.048979729413986206, 2.3156700134277344, 0.06702553480863571, -0.5602474808692932, -1.2882484197616577, 0.7287005186080933, -0.2291221022605896, -1.4793932437896729, -1.4443504810333252, 1.2869352102279663, 1.1045324802398682, 0.32981571555137634, 0.25351497530937195, 0.20931263267993927, 3.190807580947876, -1.1756609678268433, 1.0888662338256836, -0.1564209759235382, 0.9290428161621094, 1.9418201446533203, -0.7858906984329224, 1.1845957040786743, -0.9773644804954529, -1.038800597190857, 0.3505866527557373, -0.6639094352722168, -0.8641910552978516, -0.044740885496139526, -1.2225298881530762, -0.3471531271934509, -0.7031237483024597, 0.33428755402565, -0.2416241466999054, -0.050303537398576736, 1.0561333894729614], [1.172299861907959, -0.34399107098579407, -0.16672831773757935, 1.3050830364227295, 1.7004315853118896, -0.8462039232254028, -1.3005553483963013, 1.135867953300476, 2.971097230911255, 0.11133656650781631, 1.3571481704711914, -0.05841890349984169, -0.8501753211021423, 0.7980566024780273, -0.4764561057090759, 0.09536542743444443, -1.6487853527069092, -1.934863805770874, -0.8510202765464783, -0.10334821790456772, -0.666768491268158, 0.4096977710723877, -0.024919679388403893, 0.18187102675437927, -1.557234764099121, -0.543580174446106, -1.9944874048233032, -1.5291647911071777, 1.3033193349838257, 0.8579496145248413, 0.8757121562957764, 0.6267984509468079, 0.5947872996330261, 0.40533849596977234, 0.6590318083763123, 2.483701705932617, 0.7873778939247131, 0.6811739206314087, -0.35436809062957764, -0.4230524003505707, -0.6719512939453125, -0.12475855648517609, 0.6117479801177979, 0.21566176414489746, -0.9005496501922607, 0.19655534625053406, 0.1315208226442337, -0.17713861167430878, 1.8954477310180664, 0.3834853172302246, -1.338979959487915, 0.650564968585968, 0.6986250877380371, 1.3946391344070435, 1.844150185585022, -0.30338066816329956, 0.3633494973182678, 1.1171106100082397, 1.3933926820755005, -0.24074885249137878, 0.025436876341700554, -0.21291489899158478, 0.19895097613334656, 0.7250986695289612], [0.38973891735076904, -0.30385622382164, 0.04667416960000992, 0.44885122776031494, -1.0783013105392456, -1.7157880067825317, -0.05311023071408272, 1.9434480667114258, -0.9326781034469604, 0.3040600121021271, 0.6476632952690125, 0.3642381429672241, -1.3573559522628784, -0.3990243971347809, 0.20951423048973083, 0.7697287201881409, -0.013371496461331844, -1.0954492092132568, -0.34872186183929443, -1.0694653987884521, 0.19794681668281555, 0.9029959440231323, 1.2519999742507935, 0.03791928291320801, 0.7932555675506592, 1.4232107400894165, -0.5186057090759277, -0.01677876152098179, 0.6422594785690308, -0.28597399592399597, 0.8934264779090881, -0.4118524491786957, -1.2772146463394165, 0.2635899782180786, 0.7878058552742004, -0.37008973956108093, -0.1258736401796341, 0.19744732975959778, 1.6630959510803223, 1.1822247505187988, -1.714241862297058, 0.8228639960289001, -0.26184049248695374, -0.7024861574172974, -0.7184848189353943, -0.18403740227222443, -1.2502012252807617, -0.531585693359375, -1.484321117401123, -0.38809099793434143, 0.0415058434009552, 1.0584357976913452, 1.6825810670852661, 0.0885579064488411, 2.0191757678985596, 0.28591081500053406, 2.4994986057281494, -0.8769263625144958, -0.7923302054405212, -1.0083616971969604, -1.1347744464874268, 0.39281538128852844, 0.7974421381950378, 0.18542492389678955], [-0.6777889132499695, 0.06577026098966599, -0.7052571177482605, 1.305647850036621, 0.11718829721212387, -0.3495792746543884, 0.708090603351593, -0.23395244777202606, 0.8873413801193237, 0.6396963596343994, 2.0713608264923096, 1.3714016675949097, 0.3547908067703247, -0.7614237666130066, 0.14474359154701233, 2.548346757888794, -0.18905359506607056, 2.2856497764587402, 0.28157365322113037, -0.43318405747413635, 0.09040893614292145, -1.3853144645690918, 0.8378847241401672, -0.4201335310935974, 0.4579867720603943, 1.556681752204895, -0.17810764908790588, 0.09215466678142548, 1.7145088911056519, 1.5265620946884155, -1.3193405866622925, -0.1523364931344986, -0.28084468841552734, -0.11567738652229309, 1.08208429813385, 1.0772085189819336, -0.7450512647628784, -0.41781631112098694, -0.15191027522087097, 0.7166176438331604, 0.4939926862716675, 0.7910217046737671, -0.2963549792766571, 1.2163517475128174, -1.2105704545974731, -0.4044969975948334, -0.11029056459665298, 0.2938871383666992, 1.2790390253067017, 0.07596832513809204, -0.9806428551673889, 0.9636157751083374, 0.7238912582397461, 0.7718703150749207, 0.03635675087571144, 1.9854867458343506, -1.2298156023025513, 0.8176865577697754, -0.6497616171836853, -0.6886577010154724, -0.21241772174835205, -0.6652605533599854, 0.035357918590307236, -1.6545634269714355], [0.3519311547279358, 1.4957584142684937, 0.1196676641702652, 0.4572209417819977, 1.0051777362823486, 2.3002655506134033, -1.52070951461792, -0.46677619218826294, 0.8770886063575745, 1.1756324768066406, 1.0162408351898193, -1.0764431953430176, 0.8243722915649414, 0.8944731950759888, -0.21769793331623077, -0.879895806312561, 1.5137591361999512, 0.14693117141723633, 0.8273909687995911, 1.2097092866897583, -0.7873121500015259, -0.7369025945663452, -1.206864356994629, -0.30372312664985657, -0.2355739027261734, 0.48535239696502686, -0.3726915121078491, 0.20865614712238312, 0.14691343903541565, 0.2536464035511017, 0.16199176013469696, -0.09437911957502365, 0.18849186599254608, -0.8351806998252869, 0.047940172255039215, 0.6192411184310913, -2.0698559284210205, 0.07259075343608856, 1.0766513347625732, -0.9614442586898804, 0.13637655973434448, 1.2384965419769287, 0.045103877782821655, -0.561784029006958, 0.3968138098716736, -1.0752745866775513, 0.5555022954940796, 1.3093434572219849, 0.3129223585128784, -0.19317594170570374, 1.9075559377670288, -0.8953937292098999, -1.1163805723190308, 1.2654869556427002, -0.7005565166473389, 0.33622437715530396, 0.29121968150138855, 1.7809631824493408, -0.030264314264059067, -0.1761617660522461, 0.8201421499252319, 0.5326361060142517, 2.1998305320739746, 0.6942752599716187], [-0.6559233069419861, 0.9960002303123474, -1.1937521696090698, 0.496465802192688, -0.2845304012298584, -1.0231105089187622, 1.6925595998764038, 1.3249828815460205, -0.013330123387277126, -1.0263845920562744, 0.43729642033576965, -0.4423273503780365, 0.20788182318210602, 0.05026469752192497, 0.2229597121477127, -0.5451794862747192, -0.1747787892818451, -0.23196808993816376, -1.1526105403900146, -0.862949550151825, 0.001883980236016214, -1.6099141836166382, -1.0221195220947266, -2.259361743927002, 0.14642241597175598, -0.884257435798645, -0.22590801119804382, 1.230709195137024, -0.6604182720184326, 1.6458407640457153, -1.5415153503417969, 1.343664526939392, -2.568563938140869, 1.3061676025390625, -0.21084736287593842, 1.143633484840393, 1.2309592962265015, -0.1931062638759613, -0.291523814201355, 0.9841443300247192, 0.2628195881843567, -1.7502007484436035, 0.6810733079910278, -0.12976358830928802, 0.6339707970619202, -1.0282135009765625, 0.42159971594810486, 1.9646037817001343, 0.7477966547012329, 0.8511949777603149, -1.7068208456039429, 0.48118773102760315, -0.05549994856119156, -0.02556118182837963, 0.5242434144020081, -0.6082318425178528, -0.7885257601737976, -0.6643428802490234, -0.960988461971283, -0.22022968530654907, -0.3002831041812897, -1.1159443855285645, 0.5736390948295593, -0.08815096318721771], [-0.7600870728492737, 0.2900752127170563, 1.1305876970291138, -0.3473486602306366, -1.3187289237976074, 0.17934933304786682, 0.37774425745010376, -0.9267663359642029, 0.49289050698280334, 1.1475887298583984, -0.06290420144796371, -1.2129621505737305, -0.41556036472320557, 1.1472249031066895, -1.621496319770813, 0.9370229244232178, 1.6622616052627563, 0.24484165012836456, 2.12951397895813, 1.3817074298858643, -0.9347307085990906, -0.4761848747730255, 0.7407689094543457, 1.0132560729980469, -0.009097878821194172, 1.1976938247680664, 0.42181897163391113, 0.7211267948150635, -0.4565652012825012, 0.30516719818115234, 2.1473302841186523, 0.14371469616889954, -0.52846759557724, -1.210560917854309, 1.6616461277008057, -1.1614516973495483, -0.6691555380821228, -1.9000635147094727, -0.5965023040771484, 0.24803239107131958, 0.01796772889792919, 0.6331490874290466, 0.5106269121170044, 0.10350135713815689, 0.7635573148727417, -0.2520732581615448, -0.3312376141548157, 0.7607094645500183, -1.169299840927124, -0.22371768951416016, 0.8875594139099121, -0.2548205256462097, 0.5493150949478149, 0.5623875260353088, 0.714257538318634, 1.1626449823379517, -1.995010495185852, -1.9937570095062256, 1.305263876914978, -0.2864822745323181, -0.7667369842529297, -0.45766693353652954, -0.9024384021759033, -0.7264086008071899], [1.0961545705795288, 0.18155203759670258, -0.5961591601371765, -0.7284313440322876, 1.4548295736312866, -0.7282324433326721, 0.2251989096403122, 0.9645096063613892, 0.22142885625362396, 0.6692964434623718, 0.11298646777868271, -0.5760506987571716, -0.947128176689148, 0.2363402098417282, 1.6771855354309082, 2.9542653560638428, 0.5787978172302246, 1.87913978099823, -1.8454047441482544, -0.5919824242591858, 1.8947687149047852, 0.9210345149040222, -0.3663735091686249, -0.6738333702087402, 0.41910138726234436, 0.6631866693496704, -2.1573843955993652, 0.2305457442998886, 0.531855583190918, 0.2449313849210739, -1.0919150114059448, 1.2398252487182617, 0.8305686116218567, -0.07318621873855591, 1.0568435192108154, -1.308850646018982, 0.06380607932806015, -1.8762208223342896, -0.9455645680427551, -0.1474824845790863, -0.4750826358795166, 0.3966623842716217, 1.3636080026626587, 0.693135678768158, 0.3922528326511383, 0.36348971724510193, -1.550195574760437, 2.1186671257019043, 1.4400714635849, -0.03609045594930649, 0.7188137769699097, -0.028264693915843964, -0.14444500207901, -0.25933754444122314, 2.441999912261963, -1.803524374961853, 0.12541747093200684, 0.6997085809707642, 0.8706381320953369, 0.5602928400039673, 0.39945608377456665, 0.14942516386508942, -0.39907464385032654, 0.7866871356964111], [0.017505522817373276, -0.5515409708023071, 0.707484245300293, -0.5326033234596252, 0.17195795476436615, 0.548927903175354, 0.8795475959777832, -1.1698840856552124, -0.538413941860199, -0.20767000317573547, 0.2990502417087555, 0.422505646944046, 0.839093804359436, -1.0810757875442505, -0.09529154747724533, 0.6147172451019287, -0.9646248817443848, -0.5889984369277954, -0.4878126084804535, -0.13093379139900208, 0.06484673917293549, 0.842690110206604, 0.144674614071846, -0.9439247846603394, -0.22194698452949524, 1.7377420663833618, -0.5644987225532532, 2.438832998275757, -1.9633262157440186, -1.0845088958740234, 0.28095129132270813, 1.671046495437622, -1.3292285203933716, 0.38697364926338196, -2.5889501571655273, 0.018491683527827263, 0.8951234221458435, -2.228193998336792, -0.5605037212371826, 0.21175043284893036, 0.16263680160045624, -0.507629930973053, -0.6148097515106201, 0.8516418933868408, 0.29317960143089294, 0.12398026883602142, -0.5663046836853027, -0.15535585582256317, 0.9411713480949402, -0.7895097732543945, 0.47436508536338806, 0.26252374053001404, 0.6730690002441406, 0.4419296085834503, -0.5401906967163086, 1.3708570003509521, 0.8348260521888733, -0.9476884603500366, 1.0838963985443115, -0.4694913923740387, 1.1872628927230835, 0.20390412211418152, -0.670208215713501, 1.2743852138519287], [1.1621403694152832, -0.0031661419197916985, -0.34905558824539185, 0.42947307229042053, -0.9480602741241455, -2.3520090579986572, -0.3422008752822876, 1.9620320796966553, 0.5138593316078186, -0.8233914971351624, 0.4823521375656128, -2.21740460395813, -0.9933413863182068, 0.25381600856781006, 0.8773753643035889, -0.6701094508171082, -0.8699281215667725, -0.10950463265180588, 0.24483628571033478, -1.5383219718933105, -0.4372178018093109, -1.1096069812774658, 0.5371840000152588, -0.929632306098938, -1.0296168327331543, 0.01397718209773302, 0.5337637066841125, -1.0122168064117432, -0.26155492663383484, 0.8982419371604919, 1.4067859649658203, -0.1550218164920807, 1.4745820760726929, 0.05693834647536278, -0.7108479738235474, -0.9486232399940491, 0.0018549237865954638, -1.173841118812561, -0.6341996788978577, -0.26694974303245544, 1.0126614570617676, -0.32645177841186523, 0.42233404517173767, 0.720934271812439, 0.1848408579826355, -0.32310065627098083, -0.6003464460372925, -0.15573745965957642, -0.5920248031616211, 0.8150245547294617, 1.5665459632873535, 0.08569661527872086, 1.3547025918960571, -0.05534488707780838, 0.3000294268131256, -1.711859107017517, 1.6535022258758545, -0.07869880646467209, 1.201035737991333, -0.4079430103302002, -0.3373773694038391, 0.9430199265480042, 2.2643673419952393, -0.8358336687088013], [0.6114899516105652, -0.29088085889816284, -1.2826751470565796, -1.0901163816452026, -0.9801740646362305, -0.2632806599140167, 0.14269500970840454, 1.0408594608306885, 1.6863484382629395, -1.0693371295928955, -0.41086485981941223, 0.6885731816291809, 1.1189038753509521, 0.11417866498231888, 0.3229612708091736, -0.549453616142273, 2.7850284576416016, 0.5507072806358337, 2.2825539112091064, 1.14320707321167, -3.0529727935791016, 0.17530778050422668, -0.2103647142648697, -1.1413253545761108, -0.5513703227043152, 0.9084282517433167, -1.3338326215744019, 1.1680512428283691, -0.036918576806783676, -0.0070037865079939365, -0.2332351952791214, -0.10811034590005875, 0.2994857430458069, 0.21185053884983063, -0.05323922634124756, 1.5997021198272705, -0.6408903002738953, 0.3299653232097626, -0.6446132063865662, -1.3369344472885132, 2.707993507385254, 1.5961847305297852, -1.7587742805480957, -0.07938492298126221, 0.11751241981983185, 0.7265459299087524, -0.5942296385765076, 0.05789465829730034, 0.021373700350522995, -0.4460720717906952, 1.8056877851486206, 2.102973699569702, -0.09649177640676498, -1.278567910194397, 0.18803264200687408, -2.7972452640533447, 0.6820226907730103, 0.16022208333015442, 1.6279793977737427, -0.8495710492134094, -0.008446919731795788, 0.632840633392334, -0.7766054272651672, -0.3980900049209595], [-0.025450604036450386, 0.2765170931816101, -1.0382441282272339, -0.5647826194763184, -0.11190217733383179, 0.5109099745750427, -0.7580982446670532, 0.7411634922027588, 0.004762072581797838, -0.46132931113243103, 0.7118450999259949, -0.14324027299880981, 1.125937581062317, 0.2929939329624176, -0.5226160883903503, -0.5003536343574524, 0.8365471959114075, -2.290034294128418, 0.35064059495925903, 1.104135274887085, 1.1363815069198608, 0.5867191553115845, 1.194609522819519, -0.9105688333511353, 1.5656027793884277, -0.825899600982666, 0.2697126567363739, -0.12717686593532562, -0.5433887839317322, -1.1937105655670166, -0.2936939001083374, 0.08125046640634537, -0.5236230492591858, -1.0103769302368164, 0.6628184914588928, 0.06989217549562454, 0.3771282732486725, -0.2461770474910736, 0.7284214496612549, 1.632397174835205, 1.2847017049789429, -0.5122445821762085, -0.450261652469635, -1.9277185201644897, -0.3283475339412689, 0.711550235748291, -0.7093251943588257, -0.5408279299736023, -1.0413743257522583, 0.07261938601732254, -1.8657702207565308, 0.5420041084289551, 0.4713326692581177, 2.240823745727539, 0.8789844512939453, -0.6736792325973511, 0.9519534111022949, -0.935909628868103, -0.5824463367462158, -1.3743891716003418, -0.48929262161254883, -0.7718923687934875, 0.5813882946968079, 1.1308046579360962], [-0.5187490582466125, 0.16099295020103455, -0.7820347547531128, 0.640201210975647, 1.1220217943191528, 1.8524997234344482, -1.0577038526535034, 0.32882174849510193, -0.23891770839691162, 1.1977770328521729, 1.1722475290298462, -1.3390510082244873, -0.11747567355632782, 0.2579371929168701, 1.4688466787338257, -0.6381321549415588, 1.6003566980361938, -0.9758936762809753, 0.2649710178375244, -0.6045134663581848, -0.1946037858724594, -1.194217562675476, -0.3017578721046448, 1.604841709136963, 0.15917788445949554, -2.8439512252807617, 0.4311494827270508, -1.9310663938522339, 1.2580039501190186, -0.4201893210411072, 1.243571162223816, -1.2570146322250366, 1.9420890808105469, -0.40227010846138, 1.7573151588439941, 0.3511936664581299, 0.4737094044685364, 0.36468881368637085, -0.8941881656646729, -1.6662273406982422, 0.8692153692245483, 1.6883982419967651, 1.0498231649398804, 1.074070692062378, -0.6078566312789917, 3.2090513706207275, -0.5170097351074219, -0.35366588830947876, -0.38805705308914185, -0.852767288684845, -1.3793420791625977, 1.3517842292785645, 1.2892143726348877, 0.09037560969591141, -1.840989589691162, -0.25345009565353394, -0.8463606834411621, -1.2190953493118286, 0.9332318305969238, -1.2418222427368164, -0.9611207842826843, -0.9201948046684265, 0.8517559170722961, -2.767763376235962], [-0.33266544342041016, 0.3007655441761017, -1.4419440031051636, -1.6422327756881714, 1.218460202217102, -0.34289056062698364, -1.4056153297424316, 2.7317097187042236, 0.8876192569732666, 0.7383733987808228, 0.16585217416286469, -0.07026688009500504, -0.40237507224082947, -0.9963844418525696, 0.9622356295585632, -1.0517680644989014, 1.4373914003372192, 0.8218222856521606, 0.8245484232902527, -1.2058528661727905, 1.6042835712432861, -0.03069196455180645, 0.17911267280578613, -0.6296130418777466, -0.9688829779624939, -0.9158320426940918, 2.1156346797943115, 0.9136943817138672, 1.4066433906555176, -0.8199331164360046, 1.0773649215698242, 0.932947039604187, 0.6261380314826965, 0.19220879673957825, 0.2152702808380127, 0.8792867660522461, 0.678104817867279, -0.14029909670352936, -0.9010801315307617, -0.5499354004859924, -0.73863285779953, -0.24967139959335327, -0.13030290603637695, 2.391979455947876, 1.413731575012207, 0.2897886633872986, -0.22630839049816132, 1.0312532186508179, 0.3843991458415985, 0.46046844124794006, 0.3821963667869568, -0.3195227384567261, 0.3493080139160156, 0.029256194829940796, -0.027822820469737053, 0.587626576423645, 1.3741179704666138, 0.6578179597854614, 1.651168942451477, -0.10803060233592987, -1.2906806468963623, 0.02676425874233246, 1.8400593996047974, -1.1522536277770996], [1.1745409965515137, 0.19259294867515564, 0.6615788340568542, 0.47221875190734863, -1.685063123703003, 0.37978264689445496, 0.32185590267181396, -0.11409773677587509, 0.40139520168304443, -1.0172005891799927, 0.19251127541065216, 1.3730425834655762, 0.1581934690475464, 0.3371767997741699, 0.4401252269744873, 0.9504178166389465, -0.9407364130020142, -0.6879288554191589, -1.6894409656524658, -0.3404391407966614, -0.9339128136634827, -0.3755335807800293, 1.305846095085144, 1.2481852769851685, 2.533813238143921, 1.0217851400375366, -0.6624431014060974, 0.38226577639579773, 0.1591469943523407, 1.9709430932998657, 2.1507441997528076, 0.6132490634918213, -0.5344794988632202, -2.442183017730713, 1.0375858545303345, 1.247474193572998, 0.04516078531742096, -0.8417538404464722, 1.4397257566452026, -0.577518105506897, -0.07992419600486755, 0.4121582508087158, -0.1292019933462143, 0.8185630440711975, -0.5881860852241516, 0.9086071848869324, -0.8770999312400818, 1.3818411827087402, -0.4432077705860138, -0.408992201089859, -0.7648734450340271, 1.3454667329788208, 0.9489923119544983, -0.5513085126876831, -0.2836172878742218, 0.49019476771354675, -0.04223873093724251, -0.4175977408885956, 0.3680136203765869, -0.2892129719257355, -0.6503726243972778, 0.4656929075717926, -0.7027550339698792, -0.1463264375925064], [-0.3919241726398468, 0.8657063841819763, -1.7968597412109375, -0.5308741927146912, 0.5131322741508484, 0.6130601167678833, 0.15116995573043823, -0.12159469723701477, -0.5777350664138794, 0.5428760647773743, -0.6052823066711426, -0.3920575678348541, -0.7555748820304871, 0.34281811118125916, 2.525916337966919, -0.7546373009681702, -0.3749356269836426, -1.3428914546966553, 2.1299307346343994, -0.5924127697944641, 0.37791508436203003, -0.2605286240577698, 2.185687303543091, 1.0284615755081177, -0.19505231082439423, 0.48271456360816956, -1.141111969947815, -0.7554593682289124, -1.2662997245788574, -0.5477341413497925, -2.6043701171875, 0.07698163390159607, -1.4071400165557861, 0.04879789054393768, -0.06040685623884201, -0.6031877994537354, 1.9702028036117554, 0.33038631081581116, 0.4881182014942169, -1.3071558475494385, 1.0311733484268188, 0.644066333770752, -1.8293089866638184, -0.8725612759590149, 1.292816400527954, 0.25543442368507385, 1.1188675165176392, -1.6057496070861816, 1.6347538232803345, -0.9879152178764343, -1.4428956508636475, 0.9537443518638611, 1.7298184633255005, 0.3182383179664612, 2.770848274230957, -0.4314219653606415, 1.6167513132095337, -0.1707434058189392, 1.3487802743911743, 1.4229533672332764, 0.7988752722740173, 1.4514884948730469, 1.4627892971038818, -0.28070664405822754], [-0.38002994656562805, -1.1075602769851685, 1.1149662733078003, 1.1195071935653687, 1.7453866004943848, 1.036352515220642, 0.9978564381599426, 0.11874645203351974, -0.31148767471313477, 0.8028817772865295, -1.5605833530426025, -0.8270730972290039, 0.013489875011146069, 0.1748805195093155, -0.2720813453197479, 0.45002034306526184, -1.3198670148849487, -0.972233235836029, -1.0760217905044556, 0.1340213268995285, 0.7562573552131653, -1.2988367080688477, -1.707316517829895, 1.19232976436615, 0.425040602684021, 0.6248524188995361, 0.30706271529197693, -0.972548246383667, -1.3995002508163452, -1.3431602716445923, -0.051558662205934525, 0.12895971536636353, 0.2009260356426239, -0.21246673166751862, -1.0850932598114014, -0.6534494161605835, 0.4981614351272583, -1.4532312154769897, -0.3925824761390686, -0.9280907511711121, 1.151259422302246, -0.11783302575349808, -0.25952592492103577, -1.9649103879928589, 0.31297847628593445, 0.24120402336120605, 0.20831413567066193, -1.2104309797286987, 0.08196380734443665, 0.18495702743530273, 0.7018423676490784, -0.23371046781539917, 0.26985007524490356, 1.0270756483078003, -0.3336524963378906, 0.8636077642440796, 3.8707449436187744, -0.24507437646389008, -0.06744363903999329, -3.961772918701172, 0.07654871791601181, 0.3925800025463104, 0.481266587972641, 0.19805870950222015], [-2.9416632652282715, 1.598883867263794, -0.45342588424682617, 0.13456933200359344, 0.5729411244392395, 0.6447869539260864, -1.5667381286621094, -0.9147948026657104, -1.4283941984176636, 0.2689913511276245, 0.26048266887664795, -1.2778252363204956, 0.2030838429927826, 1.4852524995803833, 0.13131694495677948, -0.1021919995546341, -1.5333236455917358, -1.3879728317260742, 1.5256264209747314, -0.9391708374023438, 1.836483120918274, 0.36312827467918396, -0.9736248850822449, 0.2920311391353607, -0.3178640902042389, 0.6395013332366943, 0.18306903541088104, 1.7070732116699219, -2.2347373962402344, -0.020173843950033188, 1.0108246803283691, -0.2737414538860321, -2.141159772872925, -0.15943503379821777, 0.29603397846221924, 1.3072693347930908, 0.8520034551620483, 1.4996768236160278, -0.6641718149185181, 0.35989436507225037, 0.013628864660859108, 2.486084222793579, 0.2610114514827728, -0.2855173349380493, -0.8285161256790161, 1.71549391746521, 0.7701200246810913, -1.0047621726989746, 0.33650845289230347, -1.2493538856506348, -0.22908654808998108, 1.1967177391052246, -0.23286211490631104, -1.0885041952133179, 1.4790623188018799, 0.6531009078025818, 0.4479410946369171, 0.7573484778404236, -1.1343190670013428, 1.393951416015625, 0.386022686958313, -0.28300991654396057, 1.1389901638031006, -0.6444774866104126], [0.3902696669101715, 0.7234665155410767, 1.4713279008865356, 1.7618825435638428, 0.25900930166244507, -0.8111991882324219, -1.4330432415008545, -1.2204076051712036, 0.21315281093120575, 1.0852195024490356, 1.040515661239624, -1.0422226190567017, -0.6087555885314941, -0.6951403021812439, -1.1738721132278442, 1.5929173231124878, 0.33313655853271484, -0.23347675800323486, 0.032495543360710144, 1.2913076877593994, 0.7154372930526733, 0.8461641669273376, 0.6726300716400146, 0.4110287129878998, 0.762152373790741, 1.3646974563598633, 1.7468079328536987, -0.11230309307575226, 0.3105531930923462, 0.714770495891571, -0.5039142966270447, -0.3137929439544678, -0.025528939440846443, -0.10768377035856247, -0.6310349702835083, 0.42165282368659973, -0.9372128248214722, -3.657947540283203, 0.24629515409469604, 0.09754100441932678, 0.7215802073478699, -1.1200549602508545, 0.09993474930524826, -0.5944342017173767, -1.9358453750610352, -0.39302903413772583, -0.008769940584897995, -1.1429409980773926, 0.7632423639297485, 0.6293637752532959, -0.247099369764328, 0.9686208367347717, -0.31639882922172546, -0.6249700784683228, 0.7302671074867249, -0.696182370185852, -1.1211262941360474, 1.1611164808273315, -1.5483518838882446, 0.9895793795585632, -0.6756253242492676, 0.645783007144928, -1.2425616979599, 0.8756552934646606], [0.14599698781967163, -2.117232084274292, -0.18837870657444, 0.0859542265534401, -0.9864274263381958, 0.7934219241142273, 0.1047099158167839, -0.059350479394197464, 0.6237946152687073, -1.973281741142273, 1.2536437511444092, -0.393056720495224, -0.8686031699180603, 0.4602494239807129, -2.1794750690460205, 0.26429539918899536, -0.21599772572517395, 0.03335442394018173, -0.9738644957542419, 0.3880870044231415, -0.5446412563323975, 1.4850044250488281, 1.3345820903778076, -0.003790519433096051, -1.3695510625839233, 1.269951343536377, -0.8869873285293579, -0.2857280671596527, 0.6048699617385864, 0.14847609400749207, -1.6985079050064087, 1.2089756727218628, -1.8376696109771729, 1.4709409475326538, 0.8898923993110657, 1.1563018560409546, 0.16596044600009918, 0.4950360953807831, -1.921487808227539, -1.6410623788833618, -1.6233534812927246, -1.2549049854278564, -1.5410411357879639, 0.13523879647254944, 1.535144567489624, -0.5541127324104309, -1.0145454406738281, -0.03050766885280609, 0.48701369762420654, -1.7623080015182495, -1.2330342531204224, 0.2993617057800293, -0.8712432384490967, 1.0768423080444336, 0.6226591467857361, -2.2171928882598877, 0.24829800426959991, -0.8283294439315796, 0.8621075749397278, 0.5367662906646729, 0.6979320049285889, -1.2517156600952148, 0.1931612491607666, 1.5887123346328735], [-1.1582709550857544, -0.13761185109615326, -1.3052284717559814, 0.13760341703891754, -2.068859815597534, -0.04442480579018593, 0.01342853531241417, -0.7631001472473145, 1.4680663347244263, -0.9866296052932739, 0.42212459444999695, 0.30281177163124084, 0.5503371953964233, 0.44564205408096313, -0.04516463354229927, 0.22872298955917358, 0.787948727607727, 0.5567914247512817, 0.1106891930103302, 2.0983331203460693, -0.7739226222038269, 0.9766339063644409, -0.7876667380332947, 0.8627189993858337, -1.390106201171875, 2.252351760864258, -0.412173867225647, 2.067225456237793, 0.7375462651252747, -0.13438017666339874, 1.373445749282837, -0.47330352663993835, -0.17005300521850586, 0.8035770058631897, -1.7566347122192383, -0.010166442021727562, -0.9928602576255798, 0.5482797622680664, 1.3544074296951294, -1.9180349111557007, 1.3876097202301025, -0.20571796596050262, -0.7445986866950989, 0.012569964863359928, 1.1571115255355835, -2.511606216430664, -1.2884860038757324, 0.3125300109386444, -1.384856104850769, -0.02189948596060276, 0.11362963169813156, -0.28925421833992004, -0.8338886499404907, -0.5472193956375122, -1.0857552289962769, -0.12765194475650787, 1.192794680595398, -1.9842015504837036, -0.9737454056739807, -0.9784650206565857, -0.018354566767811775, 0.5742104649543762, 0.10582809150218964, 0.1756286323070526], [1.4515002965927124, -1.9601627588272095, -2.384521722793579, 1.547066330909729, 1.3470667600631714, 1.8546239137649536, -1.2997522354125977, 2.4488086700439453, -1.4768177270889282, 0.4164808988571167, 0.6029312014579773, -1.3832381963729858, 0.32152310013771057, 0.8842573165893555, 0.2224739044904709, 1.4736242294311523, -0.7368537783622742, -0.9751588106155396, -0.37810218334198, 1.2545216083526611, -1.1830780506134033, -0.2825610935688019, 0.7741509675979614, 1.1551851034164429, -0.22367790341377258, -0.7086623311042786, 1.0239981412887573, 1.1266669034957886, 0.12532652914524078, -1.2789280414581299, -0.86223965883255, 0.2433350533246994, 0.36155563592910767, -0.5549665093421936, -0.3846239447593689, -0.44222521781921387, 1.31044602394104, 0.6492083668708801, -0.6727436780929565, -0.30442723631858826, -0.06212250888347626, 1.7480725049972534, 1.1270180940628052, -2.4421794414520264, 0.08891872316598892, 1.0353134870529175, 0.591395914554596, -1.059780478477478, -0.07487700134515762, -0.6314916610717773, 0.6096934676170349, 1.077538251876831, -0.4892042875289917, 0.6646357774734497, 0.1462785005569458, -1.096106767654419, 1.227575421333313, -0.10221100598573685, -1.6701539754867554, -0.37054896354675293, 0.5616049766540527, 0.10710765421390533, 1.2786616086959839, -1.5299087762832642], [-1.7933968305587769, -0.09445658326148987, 1.6634601354599, -0.9463458061218262, 0.3488745093345642, 1.58207106590271, -1.5845154523849487, -0.5655264258384705, -0.6006526947021484, 0.8769561648368835, 0.017469631507992744, 0.9353804588317871, 0.0028039657045155764, -0.29448655247688293, 0.0550844706594944, 1.150500774383545, -1.7665601968765259, 0.822212815284729, 1.519686222076416, -1.7739808559417725, 0.8653499484062195, -0.15221811830997467, 0.595057487487793, -1.939620018005371, 0.2914866805076599, -0.205426424741745, -0.32500049471855164, 0.36381784081459045, 0.5111985206604004, -0.7745547294616699, 0.22427479922771454, 0.3030215799808502, 3.0238723754882812, 0.9264401197433472, 1.0909594297409058, -0.3058314025402069, 0.2315390259027481, 0.9709224700927734, -0.7609941959381104, 0.038649022579193115, -1.4636907577514648, 0.2964484691619873, 0.5364716053009033, 0.6846508383750916, 1.7629307508468628, 0.30744409561157227, 1.4966061115264893, 0.371814101934433, -0.13126114010810852, 0.5519422888755798, -0.4996025860309601, -1.3946223258972168, 1.593738317489624, 2.1992998123168945, 0.042191583663225174, -1.356589913368225, 0.389268159866333, -0.28663888573646545, 0.519057035446167, -0.3165877163410187, -0.23867665231227875, -0.6524955034255981, -0.728130042552948, -0.6225342154502869], [0.6459555625915527, -1.3534215688705444, 0.012041154317557812, -0.3485579192638397, 0.07352975755929947, 0.2567211985588074, 0.4444708824157715, 0.7558203935623169, 0.7940089702606201, -0.6831185817718506, 1.085737943649292, 0.685236930847168, 0.7438941597938538, -1.6213253736495972, 0.17696799337863922, -1.5916615724563599, 0.5599321126937866, -0.17030920088291168, 2.329322338104248, 0.14723056554794312, 0.12287440150976181, -1.0512659549713135, -0.2668623626232147, 0.5337221622467041, -0.37864431738853455, -1.6226147413253784, 0.3460475504398346, -0.26152244210243225, 1.2308932542800903, -1.1733574867248535, 0.3535102903842926, 0.6934131979942322, -0.010099370032548904, 0.11253621429204941, -0.08696816861629486, -2.795114278793335, 0.7413870692253113, 0.8783131241798401, 0.3907182812690735, 0.3935169577598572, -0.5345715880393982, 0.22561249136924744, -0.8249921202659607, -1.2038320302963257, -1.8656007051467896, 0.3480861186981201, 0.8570705652236938, 0.553397536277771, -0.4496656656265259, -0.371174156665802, 1.3223271369934082, -0.36823123693466187, 0.42248865962028503, 0.4131024181842804, -0.9453315138816833, 0.2998653054237366, -0.9299654960632324, -0.11177356541156769, -0.7473706603050232, -0.6603394150733948, 0.10112544894218445, 1.9779175519943237, -0.71994549036026, -0.19980747997760773], [-0.6853266358375549, -0.4502295255661011, -0.4953095018863678, 0.3459244966506958, 1.5591713190078735, 0.3648540675640106, 0.05536353588104248, 0.9174029231071472, -1.3599046468734741, 0.13142792880535126, -0.6125750541687012, 0.4543071985244751, 0.19101639091968536, 0.5518158674240112, 0.0011201828019693494, 0.03587445244193077, 1.1082309484481812, 0.3940027952194214, -1.3473106622695923, 1.5755057334899902, 0.9436896443367004, 0.30201753973960876, 0.5390803217887878, -1.6076217889785767, 0.32324016094207764, -0.9757506251335144, -1.5132474899291992, 0.5513666272163391, -0.09866715222597122, 2.6394150257110596, -0.9162760972976685, 0.22900594770908356, -0.34904441237449646, 1.9880423545837402, 0.6358529925346375, 0.986550509929657, -0.4155036211013794, -0.1295223832130432, -0.5968238711357117, -1.648290991783142, -0.8075034022331238, 0.003945421427488327, 0.08378955721855164, 1.4394477605819702, 1.3932037353515625, 1.394609808921814, 2.177799701690674, 1.2550008296966553, -1.9763258695602417, -3.339658260345459, 0.9173005819320679, 2.3813092708587646, 0.6701301336288452, -0.009973414242267609, -0.5540426969528198, -0.03845255821943283, -0.4179377853870392, 0.3219929039478302, -0.3103787302970886, -1.143505334854126, -0.2789473831653595, -0.24401617050170898, 2.2942864894866943, -0.3938117027282715], [-1.9928115606307983, -0.1973162591457367, -0.5906352400779724, -0.33702147006988525, 0.9633374214172363, -1.7828165292739868, -2.472071886062622, 0.23345649242401123, 0.3322828710079193, -1.84366774559021, -0.7533162236213684, 0.7397284507751465, -1.0717536211013794, -1.8362572193145752, -2.306894063949585, -0.591647744178772, 1.0534076690673828, -0.439948707818985, 1.040905475616455, 1.128499984741211, -0.33693569898605347, -1.0959385633468628, 1.3511861562728882, -1.4688646793365479, 1.0141810178756714, 0.03438173979520798, 0.10651066154241562, 0.3585711717605591, -1.2698942422866821, 0.29453045129776, 1.0279723405838013, -0.9550660252571106, -0.02763027884066105, 3.2217905521392822, -0.2814716398715973, -0.7975069284439087, 0.48627254366874695, 1.6912487745285034, -0.3701997399330139, 1.1570087671279907, 0.02126714400947094, -0.3787521719932556, -0.3039526045322418, 0.2527117431163788, -0.733807384967804, -0.3609866797924042, 0.8783220052719116, 0.5546104907989502, 0.2268332988023758, 0.3192770481109619, 0.39700746536254883, -1.172725796699524, 0.37512919306755066, -0.10431867092847824, -0.4571673572063446, -0.21474066376686096, -0.9551255106925964, 0.8758725523948669, -0.04313046485185623, 0.2560216188430786, 1.4973268508911133, 0.05556388199329376, 1.2553995847702026, -1.2671515941619873], [-0.7492642998695374, -0.33463314175605774, 2.4811172485351562, 0.045180972665548325, 1.2252602577209473, -1.776640772819519, -0.2766755223274231, 0.1538824439048767, -1.0616759061813354, 0.921029806137085, 0.9961692690849304, 1.053098201751709, 0.7555607557296753, 0.7669880390167236, -0.42194458842277527, 0.190895676612854, 0.9629760980606079, 2.238755941390991, 0.8334095478057861, -1.165605902671814, -1.32794988155365, -0.4139237403869629, -1.3363754749298096, 0.06250958889722824, 2.759401798248291, -0.48484304547309875, -1.9412691593170166, 1.3176218271255493, 0.6168611645698547, -0.2243311107158661, -1.7733919620513916, 2.616910219192505, 1.2607645988464355, 0.8365406394004822, -0.13380838930606842, -0.2181275486946106, 0.5345392823219299, -0.30308136343955994, 1.2205123901367188, -0.8456043004989624, -0.5932393074035645, -0.33459216356277466, -0.8792340159416199, -0.21991091966629028, 0.7403728365898132, -0.5135189294815063, -0.25058257579803467, 1.004860758781433, 0.10172954201698303, 1.4128448963165283, 1.922714352607727, 0.06667689234018326, 0.5013136863708496, -0.49244368076324463, -0.46286097168922424, -0.5820680856704712, -1.4426226615905762, 0.09032228589057922, -0.5707269906997681, -0.8334158658981323, -0.5499287247657776, -1.0743542909622192, 0.5358031392097473, -0.4502231478691101], [-1.0771241188049316, 0.3211165964603424, -1.8094604015350342, -0.89627605676651, -0.11377116292715073, 0.4042143225669861, -0.3278712332248688, 0.7201253175735474, -0.8398988246917725, -0.28016170859336853, -1.7366788387298584, -0.481446236371994, -1.94019615650177, 0.7838439345359802, -1.341799020767212, 1.3284529447555542, 0.5484178066253662, -1.2793666124343872, -0.6224302053451538, 0.931270956993103, -2.44931960105896, 0.8174607157707214, -0.3471832275390625, 1.1505401134490967, 1.4216859340667725, -0.13539756834506989, -0.797997772693634, 0.7765781879425049, 0.7152606844902039, 0.7611740827560425, 0.23692789673805237, 1.6994044780731201, -1.9959251880645752, -1.773777961730957, 0.2138518989086151, -0.0494418703019619, -0.45662206411361694, 0.1599370539188385, -0.07577124238014221, 0.039343152195215225, -1.006587266921997, -0.4930086135864258, -1.0224337577819824, 0.8181790113449097, -0.5136067271232605, -1.2663389444351196, 1.4116677045822144, -1.212159276008606, -0.8974413275718689, -0.40906772017478943, -0.6414170265197754, -0.004641953855752945, -0.6671927571296692, 0.7833960652351379, 1.1198291778564453, -0.0354640819132328, 0.6841519474983215, -1.2860431671142578, -1.5768611431121826, 2.266584873199463, 1.3412971496582031, -0.805777370929718, 1.2914211750030518, 0.24324677884578705], [-0.8550496697425842, -0.5903941988945007, 0.17342789471149445, 1.4293687343597412, -0.8703864216804504, 0.3775535821914673, 0.1918400228023529, 0.3328198790550232, -0.6348201036453247, 1.5102773904800415, 0.2141304463148117, 1.0510088205337524, -0.1825527846813202, 1.5745278596878052, -0.481534868478775, 0.2545015811920166, 0.5743629336357117, -0.7725797891616821, -0.8868305087089539, -0.4412372410297394, -0.5997108817100525, 0.17284953594207764, -1.846651554107666, -0.31247004866600037, 0.21151475608348846, 0.31190580129623413, -1.7633219957351685, -1.2351405620574951, -0.8117111921310425, -0.23561939597129822, -1.571947455406189, -0.7126104235649109, 0.20249414443969727, -0.7559254765510559, -0.556774914264679, 0.396151602268219, 0.11664588004350662, 0.044014573097229004, 1.1864029169082642, 2.074322462081909, -0.07772797346115112, -2.3150391578674316, -0.8175615072250366, 0.7644906044006348, -0.940267026424408, -1.8140194416046143, 1.2829738855361938, -1.159899115562439, -1.86619234085083, -1.1074037551879883, 0.23477882146835327, 0.013829787261784077, 1.5381190776824951, -0.4657486081123352, -1.0796432495117188, -1.804694652557373, -1.9948302507400513, 0.5863478779792786, 0.03087494894862175, -0.45946067571640015, -0.729321300983429, 0.2919105589389801, -0.5825885534286499, -1.4982175827026367], [0.7001764178276062, 0.22038431465625763, 0.9306591749191284, 0.047579485923051834, -0.989702582359314, -0.5326136350631714, -0.015507325530052185, 0.5458543300628662, -0.26856788992881775, 0.6282845139503479, 2.0304484367370605, -1.3398617506027222, -0.6292402148246765, 1.7489765882492065, 0.3373905420303345, 1.816921591758728, 0.4701555073261261, -0.9345023036003113, 0.8081997632980347, -0.3567979633808136, -0.302436888217926, -1.1982877254486084, 0.03308597952127457, -0.15224340558052063, -0.2596123218536377, 0.1563568115234375, 0.46914544701576233, 0.406690776348114, -0.9676939249038696, 1.912987232208252, -0.1275945007801056, -1.875712513923645, -1.6740286350250244, -0.5877782702445984, 1.1939517259597778, 0.6955888867378235, 1.4010416269302368, 0.44906240701675415, -0.4100967347621918, 1.6090574264526367, -1.5725477933883667, -0.10565061122179031, -0.7887598872184753, -0.5382025241851807, -2.7751271724700928, 0.02654714323580265, 0.6263464689254761, 0.7624571323394775, 0.9274056553840637, -0.4365471601486206, 0.16277313232421875, -0.08796429634094238, -0.4460940957069397, 0.1232050359249115, -1.2231252193450928, 0.8120162487030029, -0.6394118070602417, 0.39332112669944763, -0.8825465440750122, 0.23689810931682587, 0.34175899624824524, 2.3328018188476562, 0.9852561354637146, 0.13422347605228424], [1.011863112449646, -0.19279523193836212, -0.7974414229393005, 0.28678444027900696, -0.03182525560259819, 1.0909221172332764, 0.5635075569152832, -0.22344036400318146, 0.5588409304618835, -1.1318761110305786, 0.3472687900066376, 2.3567590713500977, 0.13674868643283844, -1.4392454624176025, 0.26354506611824036, -1.1798399686813354, -0.8307163715362549, -0.13454557955265045, -1.2519103288650513, 0.3765093684196472, -0.07112026959657669, -0.6981608271598816, -0.7629299759864807, 1.0511829853057861, -0.7350021600723267, -0.3185683786869049, -0.44490766525268555, -0.5486573576927185, 0.22634634375572205, -0.9945669770240784, -1.2680375576019287, 0.6240427494049072, -1.6139030456542969, -1.4371939897537231, -0.7100631594657898, 0.41710320115089417, 0.38656219840049744, 1.8966134786605835, -0.5328145027160645, -0.43455618619918823, 0.7554869055747986, 0.07097572088241577, -0.14640946686267853, -0.4950239062309265, 0.7782962322235107, 0.6124337315559387, 0.049281008541584015, 0.22004544734954834, 0.6950531005859375, -2.3610153198242188, 0.010052197612822056, -1.4338488578796387, 1.50154709815979, 0.2538101077079773, -0.79362553358078, 0.4965399205684662, 0.051898468285799026, -0.9234799146652222, 1.7705036401748657, -1.3913841247558594, 1.5779507160186768, -1.0380301475524902, -0.03394331410527229, -0.40493011474609375], [0.47504204511642456, -0.15270641446113586, 0.7359438538551331, -0.5160055756568909, -1.40748929977417, 0.0684337392449379, 0.10669293999671936, -0.5508250594139099, -0.06159384921193123, -0.8491747975349426, -0.16683116555213928, -0.5453557968139648, -0.461250364780426, 0.24722133576869965, 0.7096433043479919, 0.6565780639648438, 1.0573291778564453, -0.6233053207397461, -0.23250199854373932, -1.5111814737319946, 1.363362193107605, -0.7686567902565002, -0.1978045105934143, 0.4961552619934082, 0.36573392152786255, 0.5694117546081543, 0.917277991771698, -0.08984027057886124, -0.3030388057231903, -0.6733578443527222, 0.0865398645401001, 0.18791556358337402, 0.17937760055065155, 0.04799618571996689, 1.3858201503753662, 0.643205463886261, 0.6464816927909851, -0.5724501013755798, 0.7710011601448059, 1.5133507251739502, 0.48314306139945984, -0.5372262001037598, -1.0947778224945068, 0.010582608170807362, -1.4870003461837769, -1.0493297576904297, 0.6239005923271179, -0.6997150778770447, -0.39318227767944336, 0.6324121952056885, 0.5127135515213013, 1.6031146049499512, 1.7042343616485596, -0.5726732611656189, 1.376140832901001, -1.2125768661499023, 0.49293771386146545, -0.251899778842926, 0.031290020793676376, -0.0187978632748127, -1.1755869388580322, -1.3349425792694092, -1.486836314201355, 0.25577306747436523], [0.44921380281448364, 1.1827830076217651, -0.11921461671590805, -0.3480023741722107, -1.0644516944885254, 1.1910966634750366, -1.1526503562927246, 0.6532592177391052, 1.0314832925796509, -0.2170097678899765, 0.979270339012146, 0.26120421290397644, -0.5190595388412476, -0.3500550091266632, 1.256933569908142, -0.8982959389686584, 0.326029896736145, -0.031594183295965195, 1.3705933094024658, -1.2359827756881714, 1.1529725790023804, -2.017735719680786, 0.3374403715133667, 1.095263123512268, -0.5370872020721436, -0.5949376225471497, 0.83432936668396, -1.4750486612319946, -0.033435650169849396, 1.2521268129348755, -0.5706661939620972, 0.4080679416656494, -0.12987123429775238, 0.45251500606536865, -0.14889565110206604, -0.317514032125473, 1.1351605653762817, -1.2974976301193237, 2.434736490249634, 1.7414568662643433, 0.4984428286552429, -0.5540577173233032, 0.9389320015907288, -0.2724423110485077, 0.004001868888735771, -0.6139159798622131, 1.5169628858566284, 1.3477928638458252, 0.31754252314567566, -0.04082933068275452, -1.3344167470932007, 0.16895641386508942, -0.4274040162563324, 0.7118275165557861, -0.7336061596870422, -2.1160359382629395, -0.48652827739715576, 1.6364593505859375, 0.23240911960601807, -0.5108364820480347, -1.5640124082565308, 1.0042026042938232, -0.05459383502602577, 0.5465692281723022], [-0.11434142291545868, 0.42327484488487244, 0.0024884177837520838, -0.6123023629188538, -0.3800565004348755, -0.8704954981803894, -0.7109971642494202, 0.5135083198547363, -0.680696964263916, 0.810064435005188, 0.8133853077888489, 0.7714334726333618, -0.5200191140174866, 0.14364701509475708, 0.1931929737329483, -1.0389370918273926, 0.2778048515319824, -0.2034769356250763, 0.9023982882499695, -0.4144567549228668, 0.06826577335596085, 1.794384479522705, 1.5568548440933228, 0.5838010311126709, 0.8009103536605835, 1.08445143699646, 1.4698591232299805, -1.038503646850586, -1.663124442100525, -0.39867231249809265, -0.5544335246086121, 0.1617388129234314, -1.2981103658676147, -2.2909350395202637, -0.4166915714740753, 0.48431241512298584, 0.8030104637145996, -0.892173171043396, 1.8361716270446777, 0.12923437356948853, -0.31394240260124207, 0.12627576291561127, 1.2493306398391724, -0.7045771479606628, -1.559438943862915, 0.10485032200813293, 1.4737539291381836, 1.5058435201644897, -0.3258955478668213, -0.17607030272483826, -0.5211747288703918, 1.3058085441589355, 1.3078839778900146, 0.7799280285835266, -0.4528100788593292, -0.33890220522880554, -1.1904356479644775, 0.2663823068141937, 0.5212145447731018, 0.9744510054588318, -0.545773983001709, -0.5399529933929443, -1.1185640096664429, -1.3314570188522339], [0.413607656955719, -0.6868019700050354, -1.24360990524292, -1.0877025127410889, -0.8392781615257263, 0.602695643901825, -0.6402838826179504, -0.48013433814048767, -0.05013537034392357, -0.000602009822614491, -0.3773690462112427, -1.0828274488449097, 0.3661562502384186, -0.28752243518829346, 1.8660606145858765, -0.5444740056991577, 1.5320240259170532, 0.11183885484933853, 0.12113584578037262, -0.8244543671607971, 1.53806471824646, -0.25068944692611694, -1.7667704820632935, -0.1658422201871872, 0.8989765048027039, -2.717251777648926, -0.6175313591957092, 0.7673230171203613, -0.2246471345424652, -0.4124543368816376, -1.5808175802230835, -0.7429373264312744, 0.483852744102478, 1.282069206237793, -0.4741445779800415, 0.738231360912323, 0.950099527835846, -0.7555140852928162, 0.3842228949069977, 1.0165601968765259, 2.000795364379883, 0.49851715564727783, 0.3017687201499939, -0.7553201913833618, 0.895327627658844, 0.44532695412635803, 0.6501874327659607, -1.0652598142623901, 1.3332680463790894, 2.3382937908172607, -0.7279241681098938, -0.13952216506004333, -0.6444631814956665, 0.18237332999706268, 1.2816599607467651, -0.44224703311920166, -0.2776978611946106, -0.5337441563606262, -1.1430246829986572, 0.8885014057159424, 0.8313926458358765, -0.30135905742645264, -0.47757312655448914, 0.5711775422096252], [0.9232736229896545, 1.090980052947998, -0.687207818031311, 0.15426959097385406, -1.192880392074585, 1.343587040901184, -1.5353153944015503, -0.8926510214805603, -0.650784969329834, 0.5459648966789246, -0.9356480240821838, 0.26316899061203003, -0.6788514852523804, -0.39767828583717346, 0.7582010626792908, 1.458978295326233, -0.20155565440654755, 0.09254682809114456, -1.0557265281677246, 0.31564396619796753, -0.6960561275482178, 0.4380899965763092, -1.0314887762069702, -0.24876172840595245, -0.48135557770729065, 0.14526908099651337, 0.7944777607917786, -1.1820591688156128, 0.24833649396896362, -1.2404841184616089, -0.4894677698612213, 0.300657719373703, 0.3419390618801117, 0.15177108347415924, -0.8716413378715515, 0.35202571749687195, -0.027113772928714752, -0.7700632810592651, 0.1153428927063942, 1.7182039022445679, 1.4114471673965454, 0.35940083861351013, -0.649290144443512, 1.399832844734192, 1.129450798034668, 2.7647323608398438, -0.9559711217880249, -0.30869603157043457, 0.4720912277698517, 0.5590386390686035, 0.6160397529602051, -0.5425863265991211, 0.3600133955478668, 0.9748305082321167, -0.7517310976982117, -0.02524757571518421, -0.20241713523864746, -0.9167102575302124, -0.29907917976379395, -0.06896103918552399, 0.7252522706985474, -0.005669209640473127, -0.18152418732643127, 1.2894178628921509], [-0.9073283076286316, -2.4728078842163086, 1.2227927446365356, 0.566768229007721, 0.478189080953598, -0.3954693377017975, 0.7986289858818054, -1.302819848060608, 0.18990950286388397, -0.857029139995575, 0.7464311122894287, 1.3060401678085327, 2.4166719913482666, 1.0246796607971191, -0.2729751765727997, -0.05787084251642227, -0.4934711456298828, 0.5054627656936646, -1.7593176364898682, -0.8459104895591736, -0.6316466927528381, 0.6841848492622375, 0.14577023684978485, 0.6148710250854492, -0.33948320150375366, -0.494556188583374, 1.1567202806472778, 0.26142776012420654, -0.435573011636734, 1.4993377923965454, 1.0570670366287231, 1.4022798538208008, 0.9364332556724548, -1.2098236083984375, -0.7411619424819946, 0.10268440842628479, 0.7927542924880981, 0.11416938900947571, -0.058443814516067505, -0.5772785544395447, -1.6381272077560425, 0.9914586544036865, 1.1894500255584717, 0.535347044467926, -1.0336638689041138, 0.9667103886604309, -0.029080064967274666, 0.2621748745441437, -0.809533953666687, -1.0220576524734497, 0.5410036444664001, -0.10243614763021469, -1.108603596687317, -0.22187109291553497, 0.38693881034851074, -0.40302973985671997, -0.6473422050476074, -0.6889618635177612, 0.3811964690685272, 0.3061283230781555, 0.6147897243499756, -0.10378865897655487, 0.1422746181488037, -2.0593175888061523], [0.6830459833145142, -0.041603412479162216, 1.2108705043792725, -0.6866844296455383, -0.9019457697868347, 0.9777626395225525, 1.5207912921905518, -2.2307801246643066, 0.023234577849507332, -0.28832441568374634, -1.4854629039764404, -0.8440234661102295, 0.47021156549453735, -0.41813597083091736, -1.6318750381469727, -1.026014804840088, -0.6946378350257874, 1.3009175062179565, -0.4123641848564148, 1.7040420770645142, -1.0584042072296143, 1.391688346862793, 0.7996513843536377, -0.4903390407562256, -0.9473900198936462, -0.33899247646331787, 2.610952854156494, -1.7916098833084106, 0.13724204897880554, 0.232683002948761, -0.9951267838478088, -0.3774234652519226, 0.18024559319019318, 1.2554768323898315, 0.12609562277793884, -1.2280559539794922, 0.687626838684082, 0.03615008294582367, -0.7622081637382507, -0.8886703848838806, 0.42484965920448303, 1.5407675504684448, -1.2564942836761475, 0.8679261207580566, 1.8573150634765625, -0.01986604742705822, 0.7252516150474548, -0.5735288262367249, -0.2322344034910202, -0.09034054726362228, -0.41501039266586304, -1.7984591722488403, 2.354418992996216, -0.44451040029525757, -0.3132687211036682, 1.1587477922439575, 0.10338684916496277, 1.0366861820220947, 1.8521173000335693, 1.1841223239898682, 0.7132295370101929, -0.46614569425582886, -0.28391972184181213, -0.09709716588258743], [0.07304603606462479, -1.5920943021774292, -0.1074226126074791, 0.37045803666114807, -0.506637692451477, 0.5681211948394775, 2.3860690593719482, 1.312190294265747, 1.5955314636230469, 1.232338309288025, 0.021819261834025383, 1.5647640228271484, 0.8127434253692627, 2.0275039672851562, -0.4900892674922943, 1.70302152633667, 0.648804783821106, 1.5522645711898804, 0.29405543208122253, 0.3361213803291321, 0.05359293520450592, -1.2962576150894165, -0.14747358858585358, 1.8898863792419434, -0.759127676486969, -1.3334800004959106, 0.5063340067863464, -0.6797260642051697, 1.1953376531600952, -0.8214063048362732, -1.0387836694717407, -0.11837727576494217, -0.6877790093421936, -0.31495484709739685, -1.3084906339645386, -2.24365496635437, 0.5045364499092102, 0.49621742963790894, -0.8638066053390503, 1.5518732070922852, 0.3616259694099426, 1.7087677717208862, -0.8204055428504944, -0.7542726397514343, 1.0745526552200317, -0.8711925148963928, 0.029346859082579613, 0.5071378350257874, 0.5033615231513977, -0.05248641222715378, -0.23943819105625153, -0.06330349296331406, 1.3427674770355225, 0.5157254338264465, 0.6089909076690674, 1.3721261024475098, -1.4026179313659668, -0.18763253092765808, 0.11692646145820618, 0.16593946516513824, 0.9376649856567383, 1.4829972982406616, 0.5328948497772217, 0.09339027106761932], [0.22390002012252808, 0.09718763083219528, 0.44033437967300415, -0.701195240020752, 0.4402097463607788, -1.861744999885559, 1.7832671403884888, -1.182121753692627, -0.7226279973983765, 1.055666446685791, 0.6626141667366028, -0.12875722348690033, -0.7668110728263855, 0.37907397747039795, 2.3300654888153076, -0.697957456111908, 1.0363696813583374, -0.36720535159111023, 1.445213794708252, 1.3851238489151, -0.861996591091156, 0.2797835171222687, 1.1043031215667725, -0.3612465262413025, -0.05172530934214592, -0.17455428838729858, -0.41502898931503296, 0.618324339389801, 0.7648128271102905, 0.5676705837249756, 0.549699604511261, -0.6020534634590149, 0.07534866034984589, 0.45386165380477905, 0.10158636420965195, 0.44120094180107117, -0.8049624562263489, 0.049640677869319916, 0.34037983417510986, -0.12874680757522583, -0.3332251310348511, 1.1262500286102295, -0.11428426951169968, -0.1823204606771469, -0.37350839376449585, 0.7335079908370972, -1.270346760749817, 3.4721105098724365, -2.0454392433166504, 1.6130259037017822, 1.7006559371948242, 0.7756428122520447, 0.5713073015213013, 0.5562659502029419, 0.44484299421310425, -0.9079878926277161, 0.10370895266532898, -0.3677249848842621, 1.7863370180130005, 0.02109699323773384, 0.8305426239967346, -0.5280982255935669, 1.4359897375106812, -0.39874133467674255], [0.04554024338722229, 0.7970752120018005, -0.9172022938728333, 1.1379220485687256, -0.6211432218551636, 0.15044638514518738, 1.8498502969741821, 1.0311557054519653, -1.332268476486206, -0.9709996581077576, 0.3510238528251648, -0.2124066799879074, 0.9726189970970154, 0.19205090403556824, -0.04747264087200165, 0.1340552568435669, 1.7555466890335083, -0.895475447177887, 0.652830183506012, 0.8883932828903198, -0.5907351970672607, 1.4012374877929688, 0.15081198513507843, -1.7401713132858276, 0.7154646515846252, 0.5078014135360718, -0.590327799320221, -1.0914793014526367, 0.9677695631980896, 0.43632981181144714, -0.0011508847819641232, 0.059476062655448914, -0.7537795305252075, 2.029022216796875, 0.4998858869075775, 0.17880359292030334, -0.29689106345176697, 1.3951696157455444, 0.7639487385749817, 0.2485431432723999, -0.39972615242004395, -2.6188364028930664, 0.003319153795018792, 0.43221935629844666, 0.5774345993995667, 0.4203963577747345, -0.5013549327850342, 0.9322839379310608, -0.07839462161064148, -0.20452125370502472, 0.40948837995529175, -1.009858250617981, -1.6314564943313599, 0.3788681924343109, -0.13915707170963287, -0.014551203697919846, 1.8699907064437866, -0.5044600963592529, -0.5538210868835449, 1.9046847820281982, 1.6883866786956787, 0.5569279193878174, 1.039178490638733, 0.01767032779753208], [0.17020288109779358, 0.8396887183189392, -0.5314856171607971, 0.8932626247406006, 0.48151347041130066, 0.11005844175815582, 0.32400649785995483, -2.000586986541748, -0.34447601437568665, 0.5778573751449585, -0.7050409317016602, -0.3777432441711426, 1.0472290515899658, 1.0798251628875732, 1.9763940572738647, 1.754043459892273, 0.8180142641067505, -0.26435959339141846, 0.5225669145584106, 0.9302220344543457, 1.3686236143112183, -0.016010146588087082, 2.869304656982422, 0.33599546551704407, 2.154064178466797, 0.5921260118484497, -1.747960090637207, -0.6993657350540161, -1.2348676919937134, 0.12426257878541946, 0.22266651690006256, -2.718700885772705, -0.11592710018157959, 1.6071799993515015, -0.3149164021015167, 1.0856266021728516, 0.316923052072525, -2.5312867164611816, -1.0389504432678223, -0.38400837779045105, -0.8461567163467407, -1.3413395881652832, 0.21802709996700287, 0.7542001008987427, 0.23943935334682465, -0.01824808306992054, 1.736600637435913, 2.1210131645202637, -0.9374629855155945, 1.4454673528671265, -0.46559903025627136, -1.0377084016799927, 1.6950702667236328, 1.543208360671997, -0.9064232110977173, 0.9260258078575134, 0.12394949048757553, 0.47000882029533386, -0.1368129998445511, 0.5401099920272827, -0.38277581334114075, -0.5898336172103882, -0.184902161359787, 0.9387643337249756], [-0.5537378787994385, 1.3755238056182861, 0.7182478904724121, 2.147656202316284, 0.2949003279209137, -1.0946732759475708, 0.47685036063194275, 1.555914282798767, -0.11409223079681396, -2.381424903869629, 0.9303200244903564, 0.09905644506216049, -1.4961636066436768, -0.14881585538387299, -1.1990379095077515, 1.4870240688323975, -0.1102365255355835, -3.1255407333374023, -0.9999860525131226, -0.4454007148742676, -0.538089394569397, -0.12269352376461029, -0.21563509106636047, 1.2618223428726196, -0.6493690013885498, -0.5453170537948608, 1.0531562566757202, 0.18459272384643555, -1.3188161849975586, 0.02527516335248947, 0.3725261688232422, 0.12550979852676392, -1.407564401626587, -0.5625215768814087, 2.350452423095703, 1.5013443231582642, -0.01630113646388054, 0.9658581614494324, 0.22772493958473206, 0.9828701019287109, -0.2338520735502243, -0.8563164472579956, -2.0085909366607666, 0.41116493940353394, -1.7047535181045532, -0.5170707702636719, -2.276754379272461, 0.047485243529081345, 0.8889733552932739, 1.2210192680358887, 1.2331998348236084, -1.1408313512802124, 0.003919211681932211, -0.7393595576286316, 1.4047075510025024, -1.1087710857391357, 0.47885775566101074, -0.945868968963623, -0.1547859013080597, 0.5953141450881958, 0.7750325202941895, 1.6267317533493042, 0.21122433245182037, -0.23863951861858368], [2.397908926010132, 0.24308331310749054, -1.0683740377426147, 0.7644760012626648, -1.1974849700927734, 1.4306490421295166, -0.12629568576812744, -0.19225968420505524, 1.4278892278671265, -1.6803022623062134, -0.31137552857398987, -0.5447821021080017, 0.17469745874404907, -0.011295026168227196, -0.22278819978237152, 0.6712268590927124, 0.4374159276485443, 1.1347987651824951, 1.486794114112854, 0.34665077924728394, -0.19849973917007446, 0.0596051923930645, 0.23553602397441864, 0.553005039691925, 0.8314369320869446, 1.1799578666687012, 0.7039906978607178, 0.04602370783686638, -0.22650505602359772, 0.23069271445274353, 0.5984824895858765, -0.48722636699676514, -0.4821104407310486, 1.1486698389053345, 1.354830265045166, 1.2853177785873413, 0.7286930084228516, 1.7327686548233032, -0.5577349662780762, 0.4033547341823578, 0.47071632742881775, -0.9147869944572449, -1.706843376159668, 1.482851505279541, -0.416840136051178, -0.793491542339325, -0.9805527925491333, 0.5521405935287476, -2.1824283599853516, -1.57033371925354, 0.06834094971418381, -0.06304897367954254, -0.5869073867797852, 0.38692882657051086, -0.6666081547737122, -1.3791451454162598, -0.6891787052154541, -1.0609736442565918, 1.1985419988632202, -0.013499333523213863, -0.2897987365722656, -1.189290165901184, -0.0381433479487896, -1.4166395664215088], [-2.069849729537964, -0.11081856489181519, 0.9324770569801331, 0.011843875981867313, 0.05359751358628273, 1.214263916015625, 0.004771895240992308, 0.5436627268791199, -0.43452051281929016, -0.5357906818389893, -1.6544010639190674, 0.6730341911315918, 1.1361337900161743, -1.966712236404419, -1.835097312927246, -1.3620052337646484, -0.5126053094863892, -1.858028769493103, 0.8913570642471313, 1.1177204847335815, -1.0566726922988892, -0.4876672923564911, 2.099761486053467, -1.3936315774917603, -1.3990306854248047, -1.1279759407043457, 0.7782799005508423, 0.2947147786617279, -0.07429873198270798, -0.5869187712669373, -0.03845757618546486, 2.079416275024414, 1.7237046957015991, -0.050913769751787186, -2.293678045272827, 0.718021035194397, 0.3690464198589325, -2.169260263442993, -0.5563138723373413, -1.3931708335876465, -1.8538103103637695, -1.2944002151489258, -1.397239327430725, 2.0829355716705322, 0.4777185916900635, 1.0987569093704224, -0.17644567787647247, 0.06460350751876831, 1.373889684677124, 0.6580350995063782, -0.456561803817749, 0.9231430888175964, -0.07008719444274902, 1.1534109115600586, 0.6693039536476135, -0.20395992696285248, -1.2076319456100464, 0.14715610444545746, 0.18871855735778809, 1.0558967590332031, 1.1117908954620361, -1.0216944217681885, 1.2168327569961548, 1.2031763792037964], [-0.7348572015762329, -0.810545027256012, 0.47568851709365845, 0.36078187823295593, 1.1639147996902466, 0.8613399863243103, -0.9082306623458862, 1.3101565837860107, 0.04661925882101059, 0.6139427423477173, 0.9858605265617371, 1.3468774557113647, 0.9160642623901367, -1.0512245893478394, 0.34728026390075684, 1.0750164985656738, -0.7266331315040588, -1.1856101751327515, 0.5354087352752686, -0.5700519680976868, -0.79607754945755, 1.1695046424865723, -0.9468551278114319, -0.27567601203918457, -1.571051836013794, -1.2084290981292725, 0.2090258151292801, 1.1959424018859863, -1.298903465270996, -0.6839306354522705, -1.0991169214248657, 0.3173355162143707, 1.3527661561965942, 0.1487281322479248, -1.2189757823944092, -1.8838634490966797, 1.5995347499847412, 1.55580472946167, 0.5138523578643799, -1.1929593086242676, -1.0416637659072876, -1.2151790857315063, 0.6170295476913452, -0.14870736002922058, 1.2975800037384033, -2.4896275997161865, -2.1003472805023193, -0.8088928461074829, 0.18189965188503265, -1.1353329420089722, -1.7340596914291382, -0.08877421915531158, 0.11992210149765015, -1.2148915529251099, -0.6769285798072815, 2.19865345954895, -1.260191798210144, 0.31595590710639954, 2.0129311084747314, 0.4471800625324249, 0.044072896242141724, -0.6759843826293945, 0.9896212220191956, -1.0442200899124146], [1.7317451238632202, -0.5059675574302673, 1.550938606262207, -1.7031108140945435, -0.8425808548927307, 0.9117064476013184, 0.3013663589954376, 0.3325289189815521, -0.23651114106178284, -0.5698590874671936, -0.683651864528656, -0.08043057471513748, -0.6211785078048706, -0.3451998829841614, 0.4620739817619324, -0.5155826210975647, -0.46833521127700806, 0.30431559681892395, -2.8036296367645264, 0.6326196193695068, -0.43861865997314453, -0.21306683123111725, 0.058546893298625946, 0.23700818419456482, 0.6042419075965881, -1.1497728824615479, 0.1574084460735321, -0.39868709444999695, -1.3738176822662354, -0.9995784759521484, -1.2633843421936035, 0.2316470891237259, -0.3394477963447571, -0.13782790303230286, -0.3293677568435669, 0.5448741912841797, -0.5713297724723816, 0.6797986626625061, -1.5311256647109985, -1.2323100566864014, 0.13235625624656677, -0.3853975832462311, 0.7307693958282471, -1.4201639890670776, 0.30735182762145996, 0.9797067642211914, -0.5360605716705322, -0.5066474676132202, 0.6995618343353271, -1.1687406301498413, 0.43857118487358093, -0.457091361284256, 0.12266901880502701, -0.49863120913505554, -0.9408524036407471, -2.2680790424346924, 0.17390590906143188, -0.44380149245262146, 0.6741035580635071, -1.5530208349227905, 2.95910382270813, -0.9012534618377686, -0.8040966987609863, -2.0772578716278076], [-0.8510632514953613, -0.0355021171271801, -1.6843736171722412, 0.11452722549438477, -0.7057598233222961, -0.2627856731414795, 0.06982307881116867, -0.9559842944145203, 0.3605690598487854, 0.0995091125369072, 0.452274352312088, 1.0203810930252075, 0.565072774887085, 0.30815833806991577, -1.5605624914169312, 0.9200395345687866, -0.6980950832366943, -0.7226670384407043, -0.921983003616333, -0.20426516234874725, 0.48374608159065247, -1.0388470888137817, -0.5116621851921082, -0.6869874000549316, -0.5150346159934998, 0.5265124440193176, -1.764057993888855, -1.551708698272705, -1.1624653339385986, 0.6828555464744568, 0.1726289540529251, -1.376142978668213, 0.283604234457016, -0.14713765680789948, 0.14621509611606598, 0.8027926683425903, 0.4723575711250305, 2.1035263538360596, -1.495963215827942, 0.06029427796602249, 1.102536678314209, 0.9266712665557861, 0.5792344212532043, 0.7788054347038269, 1.2800629138946533, 0.33685263991355896, -2.236104965209961, 0.44295358657836914, -0.43221738934516907, 1.8032598495483398, 0.1385364532470703, 1.7036176919937134, 0.9076505899429321, -0.9533915519714355, 0.06256872415542603, -0.8160533905029297, 0.3778601288795471, 0.8942264318466187, 0.025141451507806778, 1.7439457178115845, 0.26178911328315735, 0.016865059733390808, 0.9392704963684082, -1.8117772340774536], [-1.5437846183776855, 0.15186643600463867, -1.0860004425048828, -0.6558476090431213, 0.6190698742866516, -1.0751703977584839, -1.6957460641860962, 0.05601669102907181, -0.10328284651041031, 1.5590718984603882, 0.03878169134259224, 0.34789296984672546, -1.177415132522583, 0.1636117696762085, -1.4107162952423096, 0.5850719213485718, 0.053159911185503006, 1.6389074325561523, 0.14642533659934998, -2.380305528640747, -1.3792332410812378, -0.5722951889038086, 0.026170764118433, 0.6843701601028442, -1.7267341613769531, -1.6696289777755737, -0.8170833587646484, -1.9127095937728882, 1.4215997457504272, 0.7529207468032837, 0.11694072186946869, -0.43509045243263245, -1.087950348854065, 2.0952749252319336, 0.9544336795806885, 0.9065391421318054, 0.28763049840927124, 1.3444796800613403, 0.2852286398410797, -0.6582835912704468, -0.2245495766401291, 0.3366410434246063, -0.9669746160507202, 0.38802748918533325, -1.262103796005249, 1.3231911659240723, 0.49155935645103455, 0.8465393781661987, -0.6783624291419983, -0.8781558871269226, -0.33589616417884827, -1.4862337112426758, -0.08833466470241547, -0.8389106392860413, 0.8339104652404785, -0.21791647374629974, -0.36139482259750366, 1.419556975364685, 1.0246305465698242, -1.0822207927703857, 1.2967393398284912, 0.5389677286148071, -0.7226300239562988, 1.4870991706848145], [1.774245023727417, 0.4080425500869751, -0.054942574352025986, -0.07442539185285568, 0.20155739784240723, -0.1376059651374817, 1.3697172403335571, 1.99957275390625, -0.8788243532180786, -0.663834273815155, 0.35213884711265564, 1.323641300201416, -0.22601115703582764, -0.498952180147171, -1.271968126296997, 0.4094618260860443, -0.7205702066421509, -0.7337037920951843, 1.5815366506576538, 0.6933028101921082, -1.283588171005249, 0.8122159242630005, 0.7303366661071777, -0.8389400243759155, -2.438828706741333, -0.9515416622161865, 1.6118104457855225, -0.7212370038032532, 0.9804118871688843, -0.5131032466888428, 0.7962667942047119, 1.4072059392929077, -0.45593681931495667, 0.6945666670799255, 0.8924399018287659, -0.27248328924179077, -0.37200531363487244, -2.508399486541748, 0.7305831909179688, -0.4028632342815399, -0.46030935645103455, -1.1061662435531616, 0.6067155599594116, -1.3435732126235962, 0.33534303307533264, 2.325072765350342, -0.9750669598579407, -1.1745502948760986, -1.0226348638534546, -0.07685823738574982, 0.533808171749115, 0.5714299082756042, 0.9300547242164612, 0.3778650760650635, 0.2745439410209656, -0.7717945575714111, -0.7454705238342285, 1.2669671773910522, 0.16029931604862213, -0.38820880651474, -1.799758791923523, -0.4213522672653198, -1.1027076244354248, -0.015636567026376724], [-0.7766757011413574, -0.8652307391166687, 1.3049254417419434, 1.589015245437622, -0.3967212736606598, -1.076055884361267, -1.017490267753601, 0.0306871198117733, -0.5625237822532654, -0.1807325929403305, 0.0215170718729496, 0.634822428226471, 0.11582924425601959, -0.005917731672525406, 0.8634246587753296, -0.030176393687725067, -0.48886096477508545, -0.5837053060531616, -2.485722303390503, 0.15237735211849213, -0.654425859451294, -1.7593637704849243, -0.4903137981891632, -1.2135522365570068, 0.22878427803516388, 0.15669776499271393, 0.028901323676109314, -0.9650920629501343, -1.013079047203064, -0.2254875749349594, -2.2533679008483887, -0.32879894971847534, 0.13557156920433044, 1.4605457782745361, 0.825079083442688, 0.10697957873344421, 1.2030600309371948, -0.9237022995948792, -0.9084741473197937, 1.4386497735977173, 2.015305995941162, -0.39445963501930237, -1.784682273864746, 1.3493101596832275, -0.36673784255981445, 0.7823057770729065, -0.46983054280281067, 0.9230186343193054, -0.5505471229553223, -1.2740203142166138, 0.31500911712646484, -0.8644052147865295, 1.0955548286437988, 0.655872642993927, 0.7395885586738586, -0.9563255906105042, 0.1616848260164261, -0.6815390586853027, -2.3380661010742188, 0.9771459698677063, -2.3718981742858887, -0.5053673982620239, 0.6824821829795837, 0.35776063799858093], [0.009986966848373413, 0.13871565461158752, 0.7405809760093689, -1.2320729494094849, -0.9745387434959412, 0.015782605856657028, 1.9899985790252686, 0.7819939255714417, 0.7532778382301331, -0.45098069310188293, 0.20942512154579163, 0.449144184589386, 1.0892828702926636, -1.137352466583252, -1.4095290899276733, 0.7719216346740723, -0.8154840469360352, -0.2938344180583954, 0.5524277687072754, 0.24590782821178436, -0.3794362246990204, 0.09027203172445297, -0.5265570878982544, 1.241872787475586, -0.6483054757118225, 1.3134628534317017, 0.9609651565551758, 0.872063159942627, -3.3128721714019775, 0.162641704082489, 0.41212859749794006, -0.020390411838889122, 0.7538328766822815, 0.8216601610183716, -0.7366374731063843, -1.8276278972625732, 0.3603731393814087, 0.2527294456958771, 0.4344598352909088, 0.8650617599487305, 0.5354527831077576, -1.0457470417022705, -1.1499592065811157, 0.1749308705329895, 1.2921266555786133, -2.9625940322875977, 0.8416577577590942, -1.747795581817627, 0.3119150996208191, 0.17058756947517395, 0.3943968117237091, 0.1145310178399086, -0.3377944529056549, 1.012008786201477, -1.7665002346038818, -0.3593617081642151, 1.0165066719055176, 0.19851084053516388, -0.10723084956407547, 0.934070885181427, 0.4404434561729431, -0.5159094929695129, -0.046907227486371994, -0.026949970051646233], [-0.6130313277244568, -1.182857632637024, 0.2849665880203247, -0.1132790669798851, 0.23957419395446777, -0.37511584162712097, 1.0560563802719116, 0.875013530254364, 0.7907903790473938, 0.6466787457466125, -0.9718765020370483, -1.3524073362350464, 0.021087856963276863, -0.29569926857948303, 0.6299066543579102, 0.13806624710559845, -0.7911335825920105, 0.9985134601593018, -1.1299772262573242, 0.47653838992118835, 0.7882856726646423, 0.615637481212616, 0.7571418285369873, -0.27101510763168335, 0.30527007579803467, 0.5305479168891907, -1.73429536819458, -0.7142572402954102, -0.870074450969696, -0.34187331795692444, 1.6791133880615234, 0.1509275734424591, 2.108642816543579, -0.5462481379508972, -0.8378966450691223, 2.3497800827026367, 0.5433343648910522, 0.6344207525253296, -0.5289525985717773, -0.2323979139328003, 0.2882195711135864, -1.3458294868469238, 0.4924182891845703, 0.6126905083656311, -1.9142132997512817, 0.12032562494277954, 1.261794924736023, 0.7517348527908325, -0.11213678866624832, -1.3751986026763916, 0.8449405431747437, -0.3088107407093048, 0.7376698851585388, -0.9777606129646301, 0.7536895275115967, 0.9933184385299683, -1.179250717163086, 1.0201536417007446, -1.1366053819656372, -0.09195002168416977, 2.4206149578094482, 0.7988067269325256, 0.2984699606895447, 0.03073594905436039], [1.2297027111053467, -1.4719089269638062, -1.4040472507476807, 1.798386573791504, -0.646324872970581, 0.3928397595882416, 0.09337582439184189, 0.6675066351890564, -0.8607670068740845, 0.7832455039024353, 0.42930868268013, -2.1328213214874268, -0.9508253335952759, 0.8356040716171265, -0.9665418863296509, -0.4171823263168335, -0.8247401118278503, 2.1584744453430176, 2.4157652854919434, 1.3393076658248901, 0.5996587872505188, -3.443256378173828, 1.2024036645889282, 0.9739745855331421, 0.6964760422706604, -0.30791425704956055, -0.689785897731781, -0.19632335007190704, 0.7818129062652588, -0.7376924157142639, 0.3057251572608948, 0.5151020884513855, -1.4622352123260498, -1.036076307296753, 0.9583644866943359, -0.5652883648872375, -1.2486768960952759, 0.55777907371521, -0.48385629057884216, 0.741484522819519, -1.9614994525909424, -0.5096770524978638, 0.14324046671390533, 0.3273095190525055, -0.16799187660217285, 1.074774146080017, 0.3754366338253021, 0.7417469620704651, -0.32257983088493347, -0.1735268235206604, 0.9043062329292297, 0.3463689088821411, -0.3405916392803192, -0.17215782403945923, 0.2587801516056061, 1.2035863399505615, 0.9670494794845581, -0.9455452561378479, 0.02213972806930542, 0.33975282311439514, -0.6186806559562683, 0.21072573959827423, 1.696024775505066, 0.27532249689102173], [0.3154025375843048, -1.3852524757385254, -0.2813035249710083, -0.16418829560279846, -0.5610406398773193, 0.01466516125947237, -1.1031633615493774, 1.5172213315963745, -0.3682042062282562, 1.0336107015609741, -0.015624209307134151, 0.6844319701194763, -0.9944907426834106, -0.898963987827301, -1.9805363416671753, 1.259771704673767, -0.558203399181366, 1.2431693077087402, 0.7643541097640991, -0.2691251039505005, 2.314751148223877, -0.637428879737854, 0.1257150024175644, -1.3072513341903687, -0.24222113192081451, 0.758942186832428, -0.062473732978105545, 0.9985141158103943, 1.2613199949264526, -0.3849811851978302, 0.5235677361488342, 0.8269867300987244, -1.5540611743927002, -0.3065321743488312, 0.888079822063446, 0.5047250986099243, 0.8882802128791809, -0.5773500800132751, 2.94999623298645, 1.3309491872787476, 0.028215397149324417, 0.8792386054992676, 0.001512720133177936, 1.3822576999664307, 0.6246623396873474, -1.7928673028945923, 0.42962974309921265, -0.5810983777046204, -0.9892153143882751, -0.24625849723815918, -1.7394376993179321, 1.3904991149902344, 0.72232985496521, -0.446085125207901, -1.2682124376296997, 0.10136779397726059, -0.10651199519634247, 0.14617601037025452, -0.3048803210258484, 0.8002119064331055, -0.08586779981851578, -1.3840854167938232, 2.250333070755005, 0.9904472231864929], [-1.3066837787628174, -0.14592546224594116, 0.4199676215648651, 1.8472599983215332, 0.040987804532051086, -0.26322901248931885, -1.4783438444137573, -1.5830936431884766, 3.235384702682495, 2.164140224456787, 0.423890084028244, 0.8349055647850037, 0.35707733035087585, 0.8978325128555298, -1.6652936935424805, 0.27298083901405334, -0.24505993723869324, -0.7723680138587952, -0.5864929556846619, 1.275038719177246, -0.9538895487785339, -1.3125656843185425, 0.3407087028026581, -0.026718145236372948, 1.7365905046463013, -1.404510259628296, -1.524654507637024, 0.6032196879386902, -1.2195802927017212, 0.08531396090984344, 0.23544791340827942, -1.1697887182235718, 0.6436258554458618, -0.6924763321876526, 0.24551953375339508, -0.8980957269668579, 0.6564528942108154, 0.576714038848877, -0.005704052280634642, -0.4749576151371002, 1.4575221538543701, 0.6122514605522156, -0.6738659739494324, -0.2685496211051941, 0.031128423288464546, 1.2187594175338745, -0.5124145150184631, 0.6643025875091553, -0.8635811805725098, -0.953630268573761, 0.6471469402313232, -0.17430628836154938, -1.012309193611145, 1.0994610786437988, -1.1879345178604126, 1.2183690071105957, 1.126940369606018, 1.1138968467712402, 0.015762796625494957, 0.4479786455631256, -0.384046345949173, 1.1495674848556519, -0.3567247986793518, -0.45154672861099243], [0.2022540122270584, 0.9054262638092041, 1.0176711082458496, -0.2497113049030304, 0.509073793888092, 0.8543229699134827, 0.3134556710720062, 1.5469624996185303, 1.6231695413589478, -0.6879397034645081, 1.605980396270752, 0.5407861471176147, -0.4638579785823822, -1.2943347692489624, 0.5235241055488586, 1.6211439371109009, 0.540453314781189, 1.187014102935791, 0.04314222186803818, 0.6175407767295837, 1.1875262260437012, 0.42700278759002686, -0.12671048939228058, 0.8896905183792114, 2.604639768600464, 1.0998116731643677, 0.2006986141204834, 1.012084722518921, -2.187959671020508, 0.23762664198875427, -1.9599460363388062, 0.04202791303396225, -0.2965215742588043, -2.2317731380462646, 0.690829873085022, -0.7243571877479553, 2.2592806816101074, -1.9923967123031616, 0.2721874415874481, 1.9294637441635132, 0.4550982117652893, 0.2999681830406189, 0.28831735253334045, 0.7631737589836121, -0.7260826826095581, -0.22621841728687286, -0.5780717730522156, -0.5865083336830139, -2.2202956676483154, 0.5382720828056335, -1.8607629537582397, -1.0651935338974, -2.2762935161590576, 0.4657481014728546, -0.46029388904571533, 0.8758813142776489, -1.356508731842041, -0.20562297105789185, -0.2370924949645996, 1.8219808340072632, 1.0763908624649048, -0.18206307291984558, 0.06819035857915878, -0.6522131562232971], [0.036738503724336624, -0.7975539565086365, 0.9003058075904846, -0.3915284276008606, -1.7898917198181152, -2.114518642425537, 0.7249586582183838, 0.006700790487229824, -0.4885905385017395, -0.5647625923156738, -1.096182942390442, 1.3188271522521973, 0.2372444123029709, 1.1074813604354858, -2.002305269241333, 2.399519681930542, -0.7800427675247192, 1.6903235912322998, 0.07816213369369507, 0.5620505213737488, 0.9664186239242554, -1.0972175598144531, 0.8258137702941895, 0.41335350275039673, -0.15405043959617615, 1.4311496019363403, 0.19248279929161072, -1.090149164199829, -0.20012512803077698, -1.0356624126434326, -0.6314798593521118, 1.04905366897583, 0.39261192083358765, 0.5157220363616943, -0.1875789314508438, 1.8164409399032593, 0.07470742613077164, -1.3008122444152832, -1.583453893661499, -0.19411051273345947, 1.2495923042297363, -0.9091206192970276, 0.5292970538139343, -1.2482432126998901, -1.103078842163086, -0.3389536142349243, -0.7255985736846924, 0.546886682510376, -0.7300028204917908, 1.4892908334732056, 0.03703479841351509, -0.15852123498916626, 0.39027005434036255, -0.39432162046432495, -0.13387438654899597, -0.002345700515434146, -1.1415494680404663, -0.06060483679175377, 0.7930799722671509, 1.191402554512024, 1.090289831161499, -0.8833339214324951, -1.31057870388031, 0.2515900731086731], [-1.2467877864837646, -1.154971718788147, -0.06292860209941864, -0.24290142953395844, 0.5220468640327454, 2.1045475006103516, -0.06601420044898987, -0.271998792886734, -1.1660977602005005, -1.3785234689712524, 0.5058181881904602, -0.10055623203516006, -0.5650193691253662, -0.18488940596580505, -0.5661994814872742, -1.7952710390090942, 0.14655032753944397, 0.9557843804359436, -1.6084492206573486, -0.8734754323959351, 0.45942071080207825, 1.5555672645568848, -0.5858162045478821, -0.46422791481018066, -1.5878002643585205, 0.7199185490608215, 1.3748842477798462, 0.3261321783065796, -0.9970840811729431, -0.9682623744010925, 1.8018299341201782, 0.04014318436384201, -0.44665372371673584, 0.4575679898262024, -1.1824578046798706, 0.35393473505973816, 1.6452151536941528, -0.8450599908828735, -1.6389226913452148, -1.0939866304397583, 0.42992639541625977, 0.8387766480445862, -0.948395311832428, 0.8515001535415649, 0.8337973952293396, -0.8823570013046265, 0.7795084714889526, -1.114396572113037, 0.7186616659164429, -0.693480372428894, -1.216727614402771, 0.06446438282728195, -0.4734957814216614, -0.6850650906562805, -0.8499403595924377, -0.21405164897441864, 2.353621244430542, -0.5772730708122253, 1.6606040000915527, -1.680422067642212, -1.9703963994979858, -0.33734774589538574, -0.8222916722297668, -1.191908359527588], [0.836209237575531, 0.37982749938964844, -0.6079826951026917, 0.7373717427253723, -0.9148492813110352, 0.7226287126541138, 1.2428057193756104, 1.7257499694824219, 1.2106220722198486, 0.2110367715358734, -0.8011490702629089, -1.1218883991241455, -1.2659375667572021, -0.1717035472393036, 0.41406139731407166, 0.06153539940714836, -0.31467747688293457, -0.444912314414978, 0.2652849853038788, -0.3109382688999176, -1.5209076404571533, 0.4289678633213043, -1.4926236867904663, -1.0984234809875488, -0.7668776512145996, -0.662305474281311, 0.7354984283447266, 1.1791538000106812, -1.0058348178863525, 1.373910665512085, 0.20377188920974731, -0.6602832674980164, 0.502289354801178, 0.6648336052894592, 0.10150028765201569, -0.3594728708267212, -0.6194126605987549, -0.03776377812027931, 0.9319631457328796, 2.221334934234619, -0.5411794185638428, 0.6188605427742004, -1.558647871017456, 0.17502528429031372, -1.170350193977356, -0.8477737307548523, -1.4803717136383057, 1.082674503326416, 0.08922691643238068, 1.0083154439926147, 0.5801300406455994, 0.11528851836919785, -1.195131778717041, -0.7605904936790466, -1.8165212869644165, -0.2528342008590698, 2.5129528045654297, 0.42648184299468994, 0.9714409112930298, -0.5959880948066711, 1.8412681818008423, 0.5804546475410461, -0.029966948553919792, 0.1491432785987854], [-0.8244677782058716, -0.9428097605705261, 0.859443724155426, -0.31914690136909485, 0.9416870474815369, -0.3070860803127289, -1.0936038494110107, -1.2406659126281738, -0.5051359534263611, -0.44034433364868164, 0.42809954285621643, 0.13656388223171234, -0.01370080467313528, -1.3641880750656128, 0.7132185697555542, 0.9578726291656494, 0.011416004970669746, -2.5703423023223877, 0.4387277662754059, -2.4226105213165283, -0.13998788595199585, 0.26038920879364014, -1.014767050743103, -0.030119797214865685, -0.713852047920227, -0.25517237186431885, -0.6038814783096313, -0.44219839572906494, 1.8302510976791382, -0.9694555401802063, -0.7956228256225586, -0.37670040130615234, 0.35765060782432556, 0.565342366695404, -2.3468375205993652, 0.34776321053504944, -0.13453812897205353, -0.3185402452945709, 0.9760518670082092, -0.1558191031217575, 0.0662570521235466, -1.7959822416305542, 2.0750937461853027, -1.0271258354187012, -0.5693309307098389, -0.3744106888771057, 0.7925266027450562, -0.1960151344537735, -0.40091365575790405, -0.009514220990240574, 0.49267104268074036, 0.13622500002384186, 0.341386079788208, -2.6391937732696533, 0.46531158685684204, -1.5910791158676147, -1.8494969606399536, 0.8795192241668701, 2.2397520542144775, 0.36971035599708557, 0.38250404596328735, -1.6664249897003174, -1.2897262573242188, -0.07818755507469177], [1.0315147638320923, -1.8655543327331543, 0.18364489078521729, -0.24080193042755127, -0.48505714535713196, 0.2163245528936386, 0.926785945892334, -0.3289182484149933, -2.326075315475464, 0.11057028919458389, -0.25785666704177856, -0.47571104764938354, 1.5504220724105835, -0.7841053605079651, -0.9793405532836914, -1.6391756534576416, 0.30512097477912903, 0.6921076774597168, 1.5231714248657227, -0.934499979019165, 2.211702585220337, -1.735484004020691, -1.3157426118850708, -0.5658145546913147, 0.2882622480392456, 0.2972533106803894, -0.5003327131271362, -0.8858169913291931, 0.801666796207428, 0.468589723110199, 1.4584176540374756, -0.30581510066986084, -0.4126325249671936, -0.8991366028785706, 0.5116379857063293, 1.5894695520401, 0.9028273224830627, 0.6551137566566467, 0.934148907661438, -0.15916398167610168, 0.09707777202129364, -1.9842276573181152, -0.4602981209754944, -0.19765152037143707, 0.36922839283943176, -1.0092867612838745, 0.217563658952713, -0.48680591583251953, 2.472386598587036, -0.49506890773773193, 0.35557040572166443, 0.6830130219459534, 0.4424671530723572, -0.8075917959213257, 0.648203432559967, 0.7188578844070435, 0.46503913402557373, -0.9456515908241272, -0.43365973234176636, 0.8838987350463867, -0.17081980407238007, 1.322026014328003, 1.3780897855758667, -0.5771185755729675], [1.0470194816589355, 0.7666693925857544, 0.646275520324707, -0.07001319527626038, -0.2956825792789459, -2.6522765159606934, 0.08577968180179596, -1.1868281364440918, 1.125360369682312, -1.365797996520996, 1.1057565212249756, -0.9182867407798767, 0.13236956298351288, -0.6656937003135681, -2.076014280319214, 1.295723795890808, 1.2265582084655762, 1.0302577018737793, 0.045195091515779495, 0.6609987616539001, 0.5600925087928772, -0.4367353022098541, 0.0542890690267086, 0.6668527722358704, 0.8573414087295532, 0.001616043969988823, -1.1137676239013672, -0.42570996284484863, 1.7098654508590698, -0.419099897146225, -0.8902888298034668, 0.3491983413696289, -1.5794034004211426, -0.02427501417696476, 0.03828813508152962, -0.6376284956932068, 0.5078038573265076, -2.244065999984741, -1.8873929977416992, -0.6129971146583557, -0.31255125999450684, -1.3965157270431519, 0.16841217875480652, -1.0957494974136353, 0.3482876420021057, -1.3880375623703003, -0.8358206152915955, -0.2913861572742462, -0.5103648900985718, -0.5445560812950134, -0.04831704497337341, 0.14406995475292206, -0.6109730005264282, 0.3423038721084595, 1.0831592082977295, 1.9979126453399658, 0.498915433883667, 0.10490351170301437, 0.5141571760177612, 0.5179904699325562, -0.6016175150871277, -0.8878947496414185, 1.681828260421753, -0.09041355550289154], [1.3769792318344116, -1.021893858909607, -1.6185028553009033, -0.27070438861846924, -0.08191933482885361, -2.7628228664398193, -2.3150124549865723, -0.5973808765411377, 0.09368060529232025, 0.21386858820915222, -0.9850158095359802, 1.1835808753967285, -0.3674229383468628, 0.28929078578948975, -1.5974127054214478, 1.1643680334091187, 1.9922988414764404, 0.47819197177886963, 0.739983320236206, 1.2714216709136963, 2.993903636932373, -0.32907143235206604, 0.02026408165693283, -1.3509373664855957, -1.998677134513855, 1.3748698234558105, 0.284956693649292, 0.8977475166320801, -1.1911513805389404, -0.37195274233818054, -1.48420250415802, 0.5507184267044067, 0.020786652341485023, 1.5727654695510864, 0.7337402701377869, 0.36226892471313477, -0.532799482345581, 0.07274007797241211, 0.20812779664993286, -0.96270352602005, -1.190863847732544, -0.2961556315422058, 0.24017584323883057, 1.1315802335739136, -0.12973926961421967, 1.2503529787063599, -0.7800999283790588, 1.4632631540298462, 1.8349343538284302, -0.937930703163147, -0.6142022013664246, -0.2171783745288849, -1.3914185762405396, -0.9885603785514832, 0.4631922245025635, 0.937053382396698, 0.07486389577388763, 0.6530393958091736, 0.43074989318847656, -0.18132658302783966, 1.1803224086761475, -0.6359997987747192, -1.5116934776306152, -0.739163875579834], [0.11528710275888443, 0.24264578521251678, 0.35037076473236084, -0.26824191212654114, -0.4398615062236786, 1.3392524719238281, 0.06839828938245773, -0.6222764849662781, -0.0660015419125557, 1.1125274896621704, 1.7134310007095337, -0.1367051899433136, 0.07701757550239563, 1.617142915725708, 0.7428308725357056, 0.48936179280281067, -0.5140950083732605, 0.45060575008392334, 0.06037735193967819, -1.1695470809936523, 0.6042828559875488, 0.6162793636322021, 1.0756245851516724, 0.3501940369606018, 0.28696316480636597, -0.5887038111686707, -0.8264986276626587, -1.4022873640060425, 0.1763789802789688, -0.6582477688789368, -0.493863970041275, -0.21393296122550964, 0.08267373591661453, -2.5376663208007812, 1.8449153900146484, -0.5597167015075684, 0.062018021941185, 1.5146011114120483, -0.26510441303253174, 0.8276709914207458, -0.6166942715644836, -0.16272452473640442, 0.9316585659980774, -0.3924224376678467, -0.7603952288627625, 0.4343615472316742, -0.3479526937007904, -0.8435158729553223, -0.004093915224075317, 0.6453473567962646, -1.7648574113845825, -1.1782622337341309, -0.05085984617471695, -0.6549555659294128, 1.050525188446045, -0.07925400137901306, -0.5033119916915894, -0.2656038999557495, -1.6449977159500122, 1.0065653324127197, 0.545348584651947, 1.3970763683319092, 0.37331658601760864, -1.2626153230667114], [2.079665422439575, -1.0633553266525269, 0.5201839804649353, 0.04840933158993721, -0.17970041930675507, -0.663978636264801, 0.6660364270210266, -0.3687300682067871, -2.6667799949645996, -1.1521042585372925, -0.9728092551231384, 0.7589260935783386, -0.188499316573143, 0.41998204588890076, -0.2183540165424347, -1.1723939180374146, 0.34395870566368103, 0.9360446333885193, -0.8770267963409424, -0.5744543671607971, 0.8822023868560791, 0.12392172962427139, -1.8570189476013184, -0.45230239629745483, 0.7579031586647034, -2.7318084239959717, -0.4284850060939789, -1.753200888633728, -0.5521106719970703, 0.21409974992275238, -0.2608110308647156, 0.49100053310394287, 0.25505954027175903, -0.516912579536438, 1.40471351146698, -1.59589684009552, -0.2937704026699066, 1.0809352397918701, -0.6445530652999878, -0.1860525757074356, 0.5162702798843384, -1.298696756362915, 1.6229617595672607, 0.42823079228401184, 0.6825423240661621, -0.5136150121688843, -0.08161485195159912, -0.9233823418617249, -1.4045130014419556, -0.9695830345153809, 1.5090094804763794, -1.5613958835601807, 0.28926265239715576, -0.29783275723457336, 1.3443530797958374, -0.40889570116996765, -0.2571033537387848, 1.3072317838668823, 0.2284785509109497, 0.8233695030212402, 1.043757677078247, 0.9532096982002258, 0.9256739616394043, 1.1047296524047852], [-0.7251133918762207, -0.2907539904117584, -0.5240468382835388, 0.5657871961593628, 0.16260303556919098, 1.7104994058609009, 0.3217323422431946, -0.6084064245223999, 0.9279978275299072, -0.31959471106529236, 1.3323057889938354, 0.12765462696552277, -0.4276677668094635, -0.24056239426136017, 1.1740758419036865, -1.3048080205917358, 0.05530640855431557, 0.6906406283378601, -1.2128297090530396, -2.1150941848754883, -0.9858920574188232, 1.8255277872085571, 0.14948630332946777, -0.9052876830101013, -0.7268874049186707, 0.8314362168312073, -0.19447371363639832, -0.07103028893470764, -0.9284865856170654, 0.3492892384529114, 0.09018737077713013, -1.2983425855636597, -1.531716227531433, 1.2195289134979248, -0.25799527764320374, -0.6514139175415039, -1.2977653741836548, -0.3169267773628235, 0.20316748321056366, -1.2188160419464111, -0.449200302362442, 0.6476979851722717, 1.2182968854904175, -1.0348495244979858, 1.6288236379623413, 0.3739374577999115, 1.0417263507843018, 0.2641367018222809, 1.558275818824768, 0.4561198651790619, -1.004826307296753, 0.1862657219171524, -0.3604647219181061, -1.9649789333343506, 0.2787122130393982, -0.1499653458595276, 0.2005564570426941, -1.1273319721221924, 0.6641867160797119, 0.5724023580551147, -2.0527048110961914, 0.5345333814620972, 0.31873103976249695, 0.10630020499229431], [0.4351261854171753, -0.17293211817741394, 0.4260857403278351, 1.0706720352172852, 1.0621333122253418, 0.41111254692077637, 0.8705771565437317, 1.528862476348877, -0.8830612301826477, 0.3897606432437897, 0.1261223554611206, 0.8859086036682129, 0.1120506152510643, -1.1872198581695557, -0.9997779130935669, -1.2751582860946655, 0.7387569546699524, 0.9093175530433655, -0.27668696641921997, 0.4980296790599823, -0.47515466809272766, 0.052124205976724625, -0.7770799994468689, 0.26134517788887024, -0.32781776785850525, -0.7108371257781982, -0.3477475643157959, 0.014644991606473923, 0.1420241743326187, 0.13223382830619812, 1.561912178993225, -0.5632767677307129, 0.29357075691223145, 0.45752042531967163, 0.5804264545440674, 1.1811455488204956, -0.16199612617492676, 0.8165963292121887, 2.1181650161743164, -1.354834794998169, 1.1848416328430176, -0.8276270031929016, -0.018866075202822685, 2.55255389213562, 0.3789450526237488, 0.9678850769996643, 1.431863784790039, -0.26844578981399536, -1.634001612663269, 0.9512012600898743, 0.26818323135375977, -1.2556333541870117, 0.32610926032066345, 0.8386716842651367, -0.3776266872882843, 1.5067124366760254, 0.3492897152900696, -1.0573859214782715, 0.8088139295578003, 0.4594917297363281, -0.455743670463562, 1.4673625230789185, 1.2141038179397583, 1.6931267976760864], [-1.3797045946121216, -0.7942076921463013, 0.6859723925590515, -1.2955776453018188, 0.2778523862361908, 0.560264527797699, 1.0244072675704956, 0.31418856978416443, 0.32358789443969727, -0.6321512460708618, 0.8743454217910767, -0.2565401494503021, -0.08967679738998413, -1.5856921672821045, 0.09239078313112259, 0.15499912202358246, 0.27825433015823364, 1.72794508934021, -1.0859252214431763, -1.130460500717163, -0.333344429731369, 0.5795838236808777, 1.3706328868865967, -0.8203173279762268, 0.6539497375488281, 0.5873667001724243, -0.8936034440994263, -1.0470272302627563, 1.1258935928344727, 1.4137455224990845, 1.7842482328414917, 0.9829097390174866, -1.3237230777740479, 0.11691270768642426, 0.09684910625219345, -0.2574610114097595, -0.015015054494142532, -0.24889174103736877, 0.6275366544723511, 0.7694118618965149, -0.5186207890510559, 1.315831184387207, 0.10653991252183914, -1.5128085613250732, -1.1490843296051025, 0.6077123284339905, 0.3604334592819214, -0.00622049905359745, -0.30654510855674744, -0.3788280189037323, 0.6797717809677124, 0.5214227437973022, 1.1627050638198853, 0.8396756052970886, -1.0926083326339722, -0.08312273770570755, -0.2672722339630127, 0.489289790391922, 0.03657525032758713, -0.5121028423309326, 0.9081854820251465, 0.22786587476730347, 0.21155188977718353, 0.8321160674095154], [0.27526769042015076, 0.26120761036872864, 2.1343328952789307, 0.7867581248283386, -1.2115554809570312, 2.240082263946533, -0.24763800203800201, -0.09453856945037842, 0.730920672416687, -0.006784130819141865, 0.545976459980011, 0.5547950863838196, -1.114801049232483, 1.2306597232818604, -0.7716447114944458, 1.1816952228546143, -1.0625730752944946, -1.413026213645935, -0.13697105646133423, -0.29172080755233765, -0.5672451257705688, 1.6311180591583252, -0.7742272615432739, -0.2397075891494751, -0.0020764179062098265, -0.5350795388221741, 0.6463745832443237, -1.2321950197219849, 0.6853320598602295, 0.5775143504142761, -0.09607729315757751, -1.3695600032806396, -0.7761738896369934, 0.4666370749473572, 0.9505336880683899, 0.0482492595911026, 1.6008718013763428, 1.5889360904693604, 0.49835675954818726, -1.0996471643447876, 0.0368492491543293, -2.349437952041626, -0.15201206505298615, 1.0760550498962402, -1.1721850633621216, -0.8937677145004272, 0.9286109209060669, -0.7848050594329834, -0.7960487008094788, -1.7364516258239746, -0.21796801686286926, 1.6802325248718262, 2.0842490196228027, 0.22425706684589386, 0.020469680428504944, -0.6520507335662842, -0.09982507675886154, 0.8468035459518433, 0.9424240589141846, 0.5536656379699707, 1.4125679731369019, -0.11300613731145859, 1.1599570512771606, -1.3597581386566162], [-2.318556308746338, -0.9129742383956909, -1.3802331686019897, -0.4050978124141693, 0.7068240642547607, 0.24311397969722748, 0.5687148571014404, 0.6977759003639221, 1.8065980672836304, 0.9026455283164978, -0.26919883489608765, -1.331586241722107, 0.1410730630159378, -0.36520916223526, 0.1266421675682068, 0.7041841745376587, 0.02154769003391266, -1.4889633655548096, -0.41751596331596375, -0.3777831196784973, -0.6227927207946777, 0.7939532399177551, 0.2866790294647217, 0.5836451649665833, 0.8768141865730286, -0.28314337134361267, -1.4410754442214966, 1.3112658262252808, -0.7014297246932983, -0.021504497155547142, 0.3726254999637604, 0.16882406175136566, 2.050741195678711, 1.4349218606948853, 0.17104095220565796, 1.1765069961547852, 0.30193978548049927, -1.4257173538208008, -0.09003408253192902, 1.401423692703247, 0.6787567138671875, 0.8426586389541626, 0.41430240869522095, 1.0180370807647705, 2.0971226692199707, 1.1159170866012573, -0.016656430438160896, 0.49625831842422485, -0.3718142509460449, -0.027438633143901825, 0.8655976057052612, 1.1233513355255127, 0.33261117339134216, 0.7507286667823792, 1.942023515701294, -0.7788143754005432, 0.5764836072921753, -0.24477428197860718, -1.196012258529663, -0.9238923788070679, -0.18709643185138702, 1.1434136629104614, 2.3661892414093018, -1.0383516550064087], [-0.48452576994895935, 0.34558117389678955, 0.777897298336029, 0.7597729563713074, 0.5717464685440063, 0.30010491609573364, 1.0565370321273804, -0.34709009528160095, -1.0040048360824585, -0.09902238100767136, 1.7101809978485107, -1.1803779602050781, 0.06659495830535889, 0.10338976979255676, -0.03154536336660385, -0.676994264125824, -0.4127611517906189, -1.002594232559204, 0.31792667508125305, -0.48166602849960327, -0.7577294707298279, -0.6930462121963501, -1.4855468273162842, -1.2510772943496704, -0.5342825055122375, 1.0338904857635498, 0.624649167060852, 0.033851221203804016, 0.7397890090942383, 0.31546255946159363, 0.7466850280761719, 0.14311158657073975, 1.0084154605865479, -0.1874316781759262, 1.8828808069229126, 0.010867324657738209, -0.6232537627220154, 0.4808034896850586, 0.5063899159431458, 1.3999783992767334, -0.34747761487960815, 1.210002064704895, 0.6873345375061035, 0.9262174367904663, -0.18528907001018524, -0.490016371011734, -1.136014699935913, 0.8775050640106201, -1.2470964193344116, 0.1284831166267395, 0.5555830597877502, 0.06535667926073074, -1.5490397214889526, -0.24076025187969208, 1.288210153579712, -0.5300602912902832, 2.42494535446167, -0.8028382062911987, 1.2162517309188843, 0.5284820199012756, -0.3470543920993805, -0.9009777903556824, 1.5661120414733887, 1.9507818222045898], [-0.006758600473403931, 0.07988898456096649, -1.0117429494857788, 0.2280290722846985, -0.5140944123268127, -1.9095895290374756, -1.3488218784332275, 1.4299004077911377, -0.4321400225162506, -0.8663043975830078, -0.9083707332611084, -0.7386448979377747, 0.369215726852417, -0.5292851328849792, -1.1752605438232422, 0.7846805453300476, -0.00035565425059758127, -0.42897310853004456, 0.8981987833976746, -0.6170735955238342, 0.028327710926532745, 0.5634164810180664, 0.026410352438688278, 0.4522970914840698, -0.6728962659835815, 0.9034793376922607, 0.6789065003395081, -0.762841522693634, -0.8082466721534729, 1.3443363904953003, -0.2673724591732025, -1.6894346475601196, -0.17468905448913574, 0.5259571075439453, -0.5698096752166748, 0.4055015444755554, -0.4696882665157318, 0.37291574478149414, -2.467137336730957, 0.6046625971794128, -1.565696120262146, -1.0660277605056763, -1.926421880722046, -0.671545684337616, 0.14244788885116577, 0.7351720929145813, -1.060644268989563, -0.20475181937217712, -0.44241297245025635, 0.334341436624527, 0.18671849370002747, -0.9685592651367188, -0.43963712453842163, 0.1526586413383484, 0.4660973846912384, -1.5531771183013916, 0.9607773423194885, -1.9438568353652954, 0.8473271131515503, 0.4455157518386841, 0.09606555104255676, 0.9151411652565002, -1.2954119443893433, -0.36280837655067444], [0.769321620464325, 0.8427520394325256, 0.9018202424049377, 1.063094973564148, -0.5264177918434143, -0.4364534020423889, -0.1377076655626297, -1.3641796112060547, 1.9872572422027588, -1.2928324937820435, 0.06908866763114929, -0.9146110415458679, 1.6553235054016113, 0.9158022403717041, 0.9053632616996765, -0.6242545247077942, 0.5162272453308105, 1.1451653242111206, 0.5831329822540283, -0.09496776759624481, -1.7157492637634277, 0.44642478227615356, 0.21214736998081207, 0.9685901403427124, -0.20582610368728638, 0.3896012306213379, -0.14677037298679352, -1.9552994966506958, -1.0058929920196533, -1.8545408248901367, 0.8101419806480408, -0.9929987192153931, 1.3257184028625488, 1.145372748374939, 0.7314116358757019, -1.0921015739440918, 0.4178008735179901, -1.2121860980987549, 0.12670370936393738, 0.400481253862381, -1.2322700023651123, -1.7153061628341675, 1.9445061683654785, 0.05685688927769661, -1.3789482116699219, 0.07485636323690414, 0.9338659644126892, 1.959855318069458, -0.9127357602119446, 0.1425735056400299, 0.08765476942062378, -0.7689993977546692, 0.7740767002105713, 0.27307477593421936, -1.0930906534194946, 0.579312801361084, 0.27335697412490845, -1.1691865921020508, 0.7326749563217163, 0.3017042875289917, -0.33999308943748474, 0.08190521597862244, 0.3854145407676697, 0.40386682748794556], [0.11428969353437424, -1.0395042896270752, -0.5841967463493347, 0.555411696434021, 2.0080199241638184, 0.6653932929039001, -1.0400947332382202, -0.39380714297294617, -1.052795648574829, -0.3780517280101776, -0.19992682337760925, -0.9071922302246094, -0.8584021329879761, -1.0736806392669678, 2.8716588020324707, -0.9614173173904419, 0.39314037561416626, -0.6700192093849182, -0.04941461980342865, 1.3832870721817017, 1.8018426895141602, 0.3775297999382019, 1.3867897987365723, 0.8938060402870178, -1.475173830986023, -0.8219707608222961, -0.3162711560726166, -1.858485460281372, -0.531886637210846, -0.7861508131027222, 0.4495125710964203, -1.3716011047363281, -1.3835694789886475, -1.5512440204620361, -0.8723050951957703, 0.25970375537872314, -0.5735054612159729, 0.5149999260902405, -0.6422222256660461, 0.21617794036865234, 2.3128738403320312, -0.7421908974647522, 1.8142187595367432, -0.3500478267669678, -1.2452808618545532, 0.11126220971345901, 0.3091934621334076, 1.416907787322998, 0.4931560158729553, -2.02949857711792, -0.44917821884155273, 1.4993398189544678, -0.4704505205154419, -1.319216012954712, -1.4544655084609985, -0.05700455605983734, -0.8608677983283997, 0.04899285361170769, -0.04135733097791672, 2.007340908050537, 1.3267115354537964, 0.09680631011724472, 1.1423752307891846, 0.13497482240200043], [-2.120257616043091, -1.7467765808105469, 1.2588481903076172, 0.5978323817253113, -2.0033681392669678, -1.1652964353561401, 0.23480762541294098, 1.4223860502243042, -0.2671060860157013, 0.12598241865634918, 1.1176539659500122, -0.29372891783714294, 1.0015314817428589, 0.9160459041595459, -0.5015433430671692, 0.9164558053016663, -0.22875893115997314, -1.2220447063446045, 0.5295887589454651, 1.9459843635559082, 1.2567464113235474, -0.230693057179451, -0.8603352904319763, 0.9467615485191345, 0.9644116163253784, 0.0910012274980545, 0.3954126238822937, 0.5622721910476685, -0.11649850010871887, 0.7413200736045837, -0.37247830629348755, -0.6649767756462097, -0.6453023552894592, -1.5639572143554688, -1.5883169174194336, -0.1649579554796219, -0.4114387333393097, -0.649010956287384, -1.316458821296692, -1.559671401977539, 0.6102758646011353, -0.22550149261951447, 1.8870314359664917, -0.034367986023426056, 2.6235272884368896, -1.6442524194717407, 0.5952091217041016, 0.8724886178970337, -1.3714781999588013, 0.34289947152137756, -0.6350467801094055, 0.7003810405731201, -0.3923969268798828, 0.8954657912254333, -0.07084667682647705, 1.3627580404281616, -0.02765670418739319, -2.9602794647216797, 0.4499865174293518, -0.1854458898305893, -1.4898681640625, -1.5848981142044067, -0.045671496540308, -0.20935742557048798], [0.8240178823471069, 0.03834306821227074, 1.1732673645019531, -0.8846690654754639, 0.32000285387039185, -0.2706322968006134, -0.9790268540382385, 1.806014060974121, -1.9294745922088623, -1.5065271854400635, 0.276996374130249, 0.6880154609680176, -1.4034717082977295, -2.1435546875, 0.23102134466171265, -0.012490294873714447, -0.271313339471817, 1.04600989818573, -1.5230820178985596, 0.3150896430015564, 1.010216474533081, 0.6812317967414856, -1.8327176570892334, -0.0898810550570488, -0.7079145312309265, -0.4145759642124176, -0.5214685201644897, -1.322608232498169, -0.8883654475212097, 0.7934857606887817, -0.17266137897968292, -0.7498159408569336, -0.11180701106786728, -0.9356440305709839, -1.0545034408569336, 0.7744051218032837, 3.0353763103485107, -1.869747281074524, -0.15935689210891724, 0.9217158555984497, -0.9554202556610107, -1.0451644659042358, 0.0718197226524353, -0.19870035350322723, -0.792638897895813, -0.8139969110488892, 0.11314387619495392, 0.5868731141090393, -0.22712603211402893, -0.6286288499832153, 1.0856618881225586, 0.09096544235944748, 0.8663198351860046, 0.6506568789482117, 0.9613296985626221, -1.9038690328598022, -0.4573938846588135, 0.1759387105703354, -0.3050823211669922, -1.1326063871383667, -3.2320876121520996, 1.0499178171157837, -0.25735318660736084, 0.3561916649341583], [-0.6692351698875427, 0.2940932512283325, -0.4345012903213501, 1.0539442300796509, -0.4755953252315521, 0.8636133670806885, 0.2747350335121155, -0.46101343631744385, -0.10697006434202194, -0.11014197021722794, 0.5206933617591858, 0.5413925647735596, -2.0679759979248047, -0.4437398910522461, 0.12229806184768677, 0.2967277467250824, 1.1420106887817383, 0.6209617257118225, -0.05801529437303543, -1.804997444152832, 1.3742808103561401, 0.23952911794185638, 0.7512603402137756, -0.40807849168777466, -0.763611912727356, 0.11349426954984665, 1.224623680114746, -1.059421420097351, 0.7173435091972351, -1.3812041282653809, -0.2687400281429291, -0.2628996968269348, 1.0943224430084229, -0.6811444759368896, 1.1993505954742432, -2.122755289077759, 1.6390177011489868, -0.8472884297370911, 0.6232020258903503, -0.5590147376060486, 0.9116618633270264, -0.3724824786186218, 0.05024624243378639, -0.4329981505870819, 0.5380687713623047, 0.5144686102867126, 0.8538265824317932, -0.040356241166591644, -1.1833666563034058, 0.20309296250343323, 1.3888258934020996, -0.7233291864395142, -1.8311302661895752, 0.8649401664733887, 0.6743764877319336, -0.4862060844898224, 0.6042988896369934, -0.15061154961585999, -1.505368947982788, 0.39176705479621887, 1.2323246002197266, 2.2597737312316895, 1.7088874578475952, -1.4012006521224976], [0.1511261761188507, 1.3333967924118042, 0.05419518053531647, -1.9600074291229248, 1.138433814048767, -1.4248559474945068, -0.47085297107696533, -1.0933789014816284, 0.5697622895240784, -0.8150110840797424, -0.841555655002594, -1.9537698030471802, 2.377544403076172, 0.13705433905124664, -0.7695574760437012, 0.7101126909255981, 0.0965695008635521, -1.530460238456726, 0.6316906213760376, 0.8509213924407959, -1.6119786500930786, 0.68010413646698, -1.2752656936645508, 0.1311422735452652, 0.0047713336534798145, -0.12308662384748459, 0.8044273853302002, 1.1973607540130615, 3.163583993911743, -1.436327576637268, 1.4930912256240845, 0.4226120114326477, 1.2122454643249512, -0.6460117101669312, -0.510688066482544, -2.0755155086517334, 0.5647516250610352, -0.5112701058387756, 0.3248145580291748, 1.071791172027588, -0.054047226905822754, 0.15636442601680756, 0.9102770090103149, 2.0630316734313965, 0.4998764991760254, -0.10983987897634506, 1.0437062978744507, -1.0602585077285767, -1.0729520320892334, 0.05284792557358742, 0.4146425127983093, -1.3278754949569702, -0.36564457416534424, -1.9803106784820557, 0.6096841096878052, -0.5963793396949768, -0.17924535274505615, -1.1922972202301025, 2.5217485427856445, 0.27162906527519226, 0.17641079425811768, 1.5790566205978394, 0.11638812720775604, 0.21437224745750427], [-0.3068108856678009, -0.9670699834823608, 2.729957342147827, 0.2981557548046112, 2.6700429916381836, 0.3508217930793762, 0.5441361665725708, 0.8041839599609375, 1.1666799783706665, 0.7386851906776428, 0.3990364074707031, 0.4998314678668976, 1.2956609725952148, -0.1761053055524826, 1.0984896421432495, -0.4736458957195282, -1.0485140085220337, -0.03126217797398567, -0.19355393946170807, 1.3583717346191406, 1.319711685180664, 1.3580933809280396, 1.3671982288360596, -1.9265090227127075, -0.7125245928764343, -1.7496693134307861, -0.6913748979568481, 1.2151890993118286, -1.2061762809753418, -0.18768924474716187, -1.4823689460754395, 1.7591501474380493, -1.2782469987869263, 0.5036622881889343, -0.34365227818489075, 0.7337756156921387, -1.221794605255127, 0.7487988471984863, -1.0185270309448242, 0.38733068108558655, -1.3064922094345093, 0.65447598695755, -0.9065075516700745, 1.1073638200759888, 0.941707968711853, -1.4710136651992798, -1.0333648920059204, -0.5667065978050232, 0.4651745855808258, -1.143662691116333, 0.5670187473297119, 0.1889728307723999, 0.11945009976625443, 0.6142442226409912, 0.8906870484352112, 1.0028077363967896, 0.4432870149612427, 0.17209285497665405, 0.6693239212036133, 0.727052628993988, -0.03184985741972923, -0.9820161461830139, -0.0378003790974617, 1.3543757200241089], [1.7140936851501465, -0.047263506799936295, -0.34471595287323, -0.5502427220344543, 0.9430419206619263, 0.8668965101242065, -0.4748353064060211, 0.6265760660171509, 0.16575661301612854, 0.8358585834503174, 0.27656203508377075, 1.284341812133789, -0.417398065328598, -0.26509174704551697, -0.5743510127067566, -0.9044675230979919, 0.5268896222114563, -0.13015058636665344, -1.1083242893218994, 0.35109901428222656, 1.130453109741211, -0.3950546383857727, -1.0794225931167603, -0.24375423789024353, -0.8680919408798218, -0.6094905734062195, -0.5060168504714966, 0.7761421203613281, -1.4407442808151245, -1.3361092805862427, 0.3855524957180023, 0.15471920371055603, -0.007611909881234169, -0.6721879243850708, 0.8163763284683228, -0.379878968000412, 1.5255608558654785, -0.9158158302307129, 0.6265165209770203, 0.5239344835281372, 0.512823760509491, -3.4858899116516113, 0.6750233173370361, 0.10370434075593948, -1.1034666299819946, 0.9179348349571228, 1.3125262260437012, 1.1512075662612915, -0.38437721133232117, -0.20596888661384583, -0.24778980016708374, 0.1729532778263092, 0.5931971669197083, -0.7174681425094604, 1.9419785737991333, 0.005796231795102358, -0.20839807391166687, -1.8194540739059448, -0.4635326564311981, 0.12870818376541138, 0.10029258579015732, 0.0016517355106770992, 0.280868262052536, 1.6174930334091187], [-2.3189690113067627, -2.186415195465088, 0.291942298412323, -0.20519044995307922, -0.09307466447353363, -2.160599708557129, -1.5788168907165527, -1.0025674104690552, 1.4580684900283813, -0.8152210712432861, 1.4184722900390625, 1.5334839820861816, -0.04254335165023804, -1.8825441598892212, 0.3613402843475342, -0.8187866806983948, 0.28960588574409485, -0.8740304708480835, -1.8968853950500488, -0.7224971652030945, 0.4016484320163727, -0.43507280945777893, 0.7916772961616516, 0.31490445137023926, 0.9917231202125549, -0.11303699761629105, -1.5059946775436401, -0.6199306845664978, 1.253400206565857, -0.1730085164308548, 1.1091177463531494, 0.595001757144928, 0.9769014716148376, 1.5480170249938965, -0.5860227346420288, -0.3719140291213989, 1.1859315633773804, -0.4156967103481293, 1.5940442085266113, -1.2731549739837646, -1.0459535121917725, -0.6691532731056213, -2.325218439102173, -0.5241772532463074, -1.511936902999878, -0.8659501075744629, -0.791655957698822, 0.38313058018684387, -0.7876302599906921, 0.9520806670188904, 0.5439504981040955, -0.8374037146568298, 0.5867500305175781, 1.6095597743988037, 0.586517333984375, 0.759566068649292, -2.400892972946167, 0.17576000094413757, 0.9901622533798218, 0.1930854320526123, -0.6227941513061523, 0.6988306641578674, 2.018080234527588, -0.34775498509407043], [-0.5671113729476929, -0.33812761306762695, 1.227996587753296, -1.2644416093826294, 0.7490689158439636, 0.3993656635284424, -1.2910082340240479, -1.5000821352005005, 2.5042412281036377, 0.20031146705150604, 1.12149977684021, 1.1312907934188843, 1.8264968395233154, 1.451513648033142, 0.2015255242586136, -1.972307562828064, -1.8925292491912842, -2.158304214477539, -0.034620195627212524, -0.8115357160568237, -0.8906817436218262, 0.18072476983070374, -0.4932863414287567, 2.9942080974578857, -0.729411244392395, -0.04894327372312546, 0.7480171322822571, -0.6927824020385742, -0.4791521728038788, -1.1693345308303833, -0.5245862007141113, -0.25807350873947144, -1.2099121809005737, -1.7163074016571045, -0.11609622091054916, -1.1146482229232788, 1.4888211488723755, -0.4206528961658478, -0.631629228591919, 0.2428281605243683, -0.45642298460006714, 0.5601186752319336, 0.1570250242948532, -0.04955441877245903, -0.9002982974052429, 1.5861183404922485, 0.3242793679237366, -1.2913779020309448, -0.5812317728996277, -0.9994441270828247, 0.7190020680427551, -0.4777832627296448, 0.23314431309700012, -0.37916940450668335, -1.5119473934173584, 1.721495270729065, -1.8323856592178345, -1.893195390701294, -0.8107333779335022, -1.8303292989730835, -1.041014552116394, -1.2481839656829834, -0.9776340126991272, -0.5518083572387695], [2.323988437652588, -0.31006893515586853, -0.931644082069397, 0.7906094193458557, 0.029677141457796097, -1.5357568264007568, 1.440979242324829, 0.36474597454071045, -0.18573403358459473, -0.5693661570549011, 1.0916016101837158, -1.1103907823562622, -0.20697927474975586, 0.10304645448923111, -0.12181223928928375, -0.5751471519470215, -0.1547023057937622, 0.6805568933486938, 0.708385705947876, -0.2131829857826233, -1.2433888912200928, 0.9322186708450317, -0.4664548635482788, -1.0511349439620972, -0.7003458738327026, -0.3022143244743347, 0.0891902968287468, 1.9562567472457886, -0.7755669951438904, -0.6071190237998962, -2.7326996326446533, 1.1401692628860474, 0.601789116859436, 1.3284355401992798, 1.4171254634857178, 0.9455412030220032, -1.6715805530548096, -2.2229809761047363, 1.1020848751068115, -0.12726540863513947, 0.7479447722434998, -0.28347060084342957, -1.7717978954315186, 0.4719947874546051, 1.3643990755081177, 2.8091015815734863, 0.734361469745636, 0.07443148642778397, -0.1309436559677124, -1.2492039203643799, 0.17863430082798004, 0.2999723255634308, 1.0500060319900513, 1.3974075317382812, -1.372988224029541, -0.443618506193161, -0.01201343722641468, -0.28020116686820984, -0.13734138011932373, 1.0040992498397827, 0.13493919372558594, 0.7791578769683838, 0.11711908876895905, -0.8015828132629395], [0.7413521409034729, -0.9365488290786743, 0.2152567207813263, 0.4422163665294647, -0.23713642358779907, 0.14826132357120514, -0.40134525299072266, -0.43459025025367737, 0.05655420944094658, 0.8822526335716248, -0.7459632158279419, 0.4957219064235687, -0.7227190136909485, 0.9181436896324158, 1.7581825256347656, 0.3704642951488495, 1.4638671875, 0.5234057307243347, -1.0474530458450317, 1.3376272916793823, 0.5759413838386536, -0.9457723498344421, 0.19329912960529327, -0.41589438915252686, -1.191932201385498, -0.8739111423492432, -0.12781614065170288, -0.5032938718795776, 0.7757406234741211, -0.9295819997787476, -1.3503851890563965, 0.5819727182388306, 0.8167422413825989, -0.2450125813484192, -3.223174571990967, 1.3393347263336182, -0.9785149693489075, -1.8497450351715088, -0.06596031785011292, -0.016843272373080254, 0.8186144828796387, -1.457880973815918, -1.5481631755828857, -0.05945318192243576, 1.1270482540130615, -0.2987653315067291, 1.2067335844039917, -1.2938497066497803, -0.004703032784163952, -0.12308430671691895, 0.3056875169277191, 0.2191474288702011, 1.4476763010025024, -1.221408724784851, 2.295612096786499, 0.48665693402290344, 0.5977540612220764, 0.06135086342692375, 0.047520726919174194, 1.1761399507522583, 0.7126040458679199, 1.2065776586532593, 0.7904230952262878, -0.5104968547821045], [1.0470390319824219, 0.5020803213119507, -0.47824060916900635, -1.1839783191680908, -1.6135826110839844, 0.695559024810791, -0.19869312644004822, -0.5836672186851501, -0.48046958446502686, -0.1455506533384323, 0.14704789221286774, 0.5846211314201355, -0.09310010820627213, -1.114781379699707, -1.0386298894882202, 0.5260425806045532, 0.5482217073440552, -2.6743881702423096, 0.28070831298828125, 1.2989811897277832, -1.8709912300109863, 0.045025262981653214, 0.050664301961660385, -1.6663554906845093, -0.47754257917404175, -0.2824273109436035, -1.4749135971069336, -0.5621592998504639, 0.28504806756973267, 1.2447172403335571, -1.3248803615570068, 1.0761977434158325, -1.3108261823654175, 0.2719344198703766, -1.3712862730026245, -0.6045010685920715, -0.9487612843513489, 1.0838677883148193, 1.7535269260406494, 0.29927849769592285, -0.5239101648330688, -1.2939581871032715, -1.2201355695724487, 2.021885871887207, 0.036684706807136536, 0.7869813442230225, 0.4287198781967163, 0.4527312219142914, 1.8417940139770508, 0.4780997633934021, 0.8607381582260132, -0.3124389052391052, -0.37755414843559265, -0.4408054053783417, 0.05587359145283699, -0.18511024117469788, 0.5164368152618408, 1.4241828918457031, 0.4636109173297882, -0.12577170133590698, 0.7555304765701294, -0.43920812010765076, -1.671790599822998, 0.1052277609705925], [-1.9321976900100708, -1.6439601182937622, -0.38014790415763855, 0.7036485075950623, 0.9801093935966492, -1.266305685043335, 0.26868492364883423, 1.4952759742736816, -0.34363844990730286, -0.40610313415527344, 0.2883872985839844, -0.2844681143760681, -0.8221792578697205, -0.05621354281902313, 0.7127137184143066, 0.45532578229904175, -1.1017675399780273, 1.2064623832702637, 0.411687970161438, -1.6669083833694458, 1.3622276782989502, -1.023532509803772, 0.43028494715690613, -1.0373108386993408, 0.7906435132026672, -1.4712493419647217, 0.0015603351639583707, -0.14497119188308716, 0.0586804524064064, -0.9812502861022949, -0.33087483048439026, -0.7768436670303345, -2.4843344688415527, -0.13928104937076569, -0.268151193857193, 0.6286907196044922, -0.5513648390769958, 0.47349807620048523, 0.9248313307762146, -2.1141574382781982, -0.07639893889427185, 0.351235955953598, -0.7003654837608337, 0.6563951969146729, -0.041910383850336075, -0.32052943110466003, -0.5639033913612366, 2.2668888568878174, -1.1842259168624878, 0.40535297989845276, 1.5423977375030518, 1.1709975004196167, -1.0780775547027588, -0.03470776230096817, -0.43296700716018677, 0.2379254847764969, -0.36604073643684387, 0.9341073036193848, -1.6202605962753296, 1.817152976989746, -0.9293754696846008, -0.22263649106025696, 1.3183602094650269, 0.9762653708457947], [-0.47409096360206604, -0.12531757354736328, -0.8057257533073425, -0.7224950790405273, 0.12838114798069, -0.3001137673854828, -0.6587826609611511, -0.3222672939300537, 0.5604296326637268, 0.3102776110172272, -2.06042218208313, 0.4044986665248871, -1.2162541151046753, -1.202383041381836, 0.05884183570742607, -0.7914909720420837, 0.9868341088294983, -1.4771116971969604, -0.44842439889907837, -2.624253749847412, -2.237459659576416, 0.20937740802764893, -1.2205945253372192, -0.12440677732229233, -0.47876811027526855, -1.179457426071167, -0.29269030690193176, -1.6519708633422852, -0.2109607309103012, -0.26060160994529724, 0.10914382338523865, -0.0046625942923128605, 1.298764705657959, 1.297667384147644, 0.6981869339942932, -0.9985934495925903, 0.04713650420308113, 0.8444066047668457, -0.3068094551563263, 0.7577964067459106, -0.5365545153617859, -1.5074925422668457, 1.2354600429534912, -0.01386727299541235, -1.3680814504623413, -0.5438335537910461, 0.8671127557754517, 0.529152512550354, -1.2692657709121704, 2.3963780403137207, 0.20390988886356354, 1.693482756614685, 0.5176936984062195, -0.6685184240341187, 1.4152835607528687, 0.08577179163694382, 0.21086663007736206, -0.5793246626853943, -0.20419037342071533, 0.28692618012428284, -0.39866065979003906, 0.5243633389472961, 0.9134643077850342, -1.5830304622650146], [2.832730293273926, -0.26654964685440063, 0.24967172741889954, 1.0408748388290405, 0.4056878089904785, -2.7987632751464844, -0.056243859231472015, -0.44144406914711, 0.7046980261802673, -0.30977317690849304, 0.23994536697864532, -1.229224443435669, 0.13679076731204987, 0.8522369861602783, -1.4372178316116333, -0.6891822218894958, -0.8889493942260742, -0.8967165350914001, -0.25366857647895813, -0.8091720342636108, -1.7838771343231201, 1.0434749126434326, 0.8517952561378479, 1.158718228340149, 0.8251698017120361, 0.21051722764968872, 0.29267147183418274, -0.39550310373306274, -1.6232187747955322, -1.0749481916427612, 1.672324776649475, 0.6304285526275635, 2.101170539855957, -0.33382588624954224, -1.1791455745697021, -0.7838903069496155, -0.8955071568489075, 0.8897022604942322, 0.8388388752937317, 2.159860372543335, 1.217755675315857, 0.7770653963088989, 0.08688440173864365, 0.6760127544403076, 1.9483250379562378, -0.36069318652153015, 1.5570075511932373, -0.8998899459838867, -1.2626268863677979, -0.25094419717788696, 0.46137967705726624, 1.5227802991867065, 2.1199779510498047, -0.7319544553756714, -0.6898360252380371, -0.9107262492179871, 1.4716413021087646, 0.623447597026825, 0.04775644838809967, 0.3648534417152405, -1.457537055015564, 2.143214225769043, 0.1321132481098175, -0.4273122549057007], [0.1618054062128067, 1.4313071966171265, 0.6859964728355408, 0.6074438095092773, -0.09348975121974945, -0.8459326028823853, 1.9623976945877075, -0.9748777151107788, 0.44396814703941345, 0.7070522308349609, 2.1932179927825928, -0.7586981654167175, 1.8570082187652588, -0.22661110758781433, -0.5294385552406311, -1.4798561334609985, 1.5631591081619263, 0.7294934391975403, 0.5289046764373779, -0.6813281774520874, 0.7842283248901367, 0.08777475357055664, -0.014216787181794643, 0.2898566424846649, 0.30908796191215515, -1.1292177438735962, -0.26289045810699463, 1.6607955694198608, -0.5303741097450256, -1.0810190439224243, 0.17519059777259827, -0.7811893820762634, 0.7240387797355652, -1.9184789657592773, 1.8232332468032837, -2.640580654144287, -0.18248924612998962, -0.03458767384290695, -1.5100537538528442, -0.058875907212495804, -0.32027366757392883, -1.234054684638977, 1.272515892982483, -1.454294204711914, -1.0999491214752197, -0.3216632902622223, -2.0342843532562256, 0.05147096887230873, -0.32900702953338623, 1.2542022466659546, 0.014497530646622181, -0.712615430355072, 1.2582290172576904, -0.9345847368240356, 0.2731851041316986, -0.08173838257789612, 1.2395519018173218, 1.7937321662902832, -0.11864325404167175, -0.7230169177055359, -0.7241101264953613, 0.8023566603660583, -1.6016486883163452, 0.8137636780738831], [1.3390592336654663, 0.6240484118461609, 0.34751927852630615, -0.2372599095106125, 1.211877465248108, -0.8577957153320312, -0.2721754312515259, 0.6849236488342285, -0.09310808777809143, -0.8636417388916016, -1.1498523950576782, 0.0914691835641861, 0.28246963024139404, -0.4041658043861389, 1.3744380474090576, -0.4103774428367615, -0.6810664534568787, 0.29744598269462585, 2.311110734939575, 0.7302883267402649, -1.7035878896713257, 2.5696933269500732, -0.18154072761535645, -0.8012884855270386, 0.4946949779987335, -0.9131307005882263, -0.9836649298667908, 0.16605614125728607, -0.8496171236038208, 0.09729057550430298, 0.799751877784729, 2.143842935562134, -0.6102415919303894, 0.9855457544326782, -0.480116605758667, -0.5048865675926208, -1.006151556968689, 0.5018129348754883, 0.33241599798202515, 1.2142839431762695, 1.057194709777832, 0.8865087032318115, -0.06688588857650757, -0.18949154019355774, -0.912738025188446, 0.8927290439605713, 0.15711720287799835, -1.1454111337661743, 1.5673439502716064, 0.9230190515518188, 1.1637897491455078, 0.17654544115066528, -0.3890026807785034, 1.0626282691955566, 0.6834403276443481, -0.4357864260673523, -1.8785040378570557, -2.0805280208587646, -1.362335443496704, 0.3193295896053314, -0.17803052067756653, 0.3943576216697693, 0.19374454021453857, 1.6035314798355103], [-1.0034873485565186, 0.3182479441165924, -0.6372718214988708, -0.367419958114624, -0.32743969559669495, -0.1781589537858963, -0.653412938117981, -1.572896957397461, 0.3540322482585907, 0.5972811579704285, -0.6760153770446777, -1.3132038116455078, 0.7172794938087463, 1.02496337890625, 1.0087544918060303, -1.512002944946289, -0.10248441249132156, -0.10396832972764969, -0.5544435381889343, 1.498814344406128, 0.2525682747364044, 0.9694531559944153, -3.0737178325653076, -0.14157690107822418, 1.0439952611923218, -0.04071088880300522, 0.722317099571228, -0.08914553374052048, 2.2741403579711914, -0.8848179578781128, -0.37016692757606506, 0.06612654775381088, 0.31819745898246765, 0.9933125972747803, -0.36531975865364075, 1.2641593217849731, 0.29425182938575745, -0.26149237155914307, 0.5089092254638672, -0.20827315747737885, -0.9616403579711914, 0.7736259698867798, -0.28124305605888367, -0.0026624843012541533, -0.22168368101119995, -0.2196478396654129, -2.076897621154785, -0.17222514748573303, -0.8953956365585327, -0.5159684419631958, 0.4856012761592865, -0.2952130138874054, 1.3992810249328613, 0.263386994600296, 0.1708317995071411, -1.3073461055755615, 0.24887506663799286, -1.7980539798736572, -0.10450475662946701, -1.2034238576889038, -0.21031327545642853, -2.941997528076172, 0.2713731825351715, 0.8046132922172546], [-2.979595899581909, -1.7882169485092163, -0.529788613319397, -0.7211509943008423, -1.7773792743682861, 0.19616971909999847, -0.17421230673789978, -0.799625813961029, -1.5788178443908691, -0.5597314238548279, -0.24840207397937775, -0.7247905135154724, 2.258718490600586, -0.678698718547821, -0.7732076644897461, -0.11783842742443085, 1.613338589668274, 0.9628666639328003, 1.5929206609725952, 0.26702946424484253, 0.7474461793899536, 0.21064285933971405, 0.11017316579818726, 0.19931361079216003, 0.8269401788711548, -0.49959155917167664, -0.7613434791564941, 2.761995315551758, -1.6307878494262695, 0.05085340142250061, -0.5877026319503784, 1.1272133588790894, -1.6169731616973877, 1.7734488248825073, -0.09090199321508408, -0.037569329142570496, 0.0068962471559643745, -1.821911334991455, -1.1872180700302124, 0.9840044975280762, 0.13731616735458374, -0.14671590924263, -1.6286516189575195, 0.02109598182141781, 1.616066336631775, -1.1112171411514282, 0.08328651636838913, 0.7324436902999878, -0.8574046492576599, 0.4299517869949341, 1.3608241081237793, 0.7814388871192932, 0.10967642813920975, -1.1007682085037231, 1.1790082454681396, 0.48542317748069763, 0.3727152645587921, 0.5852279663085938, -1.8201032876968384, 0.4074830710887909, 0.10727771371603012, -0.3538644313812256, 0.07134118676185608, 0.8197282552719116], [0.5850857496261597, 0.8610619306564331, 0.0670689046382904, -0.5327554941177368, 1.3798727989196777, -0.18014733493328094, -0.6327736377716064, 0.7157217860221863, -0.8797940611839294, 0.6315007209777832, 0.5082845091819763, 0.2352524846792221, -1.5129984617233276, -0.09501951932907104, -0.40819770097732544, 1.0845623016357422, 0.31087011098861694, 1.2264388799667358, 0.13427706062793732, -0.047162435948848724, 0.13386018574237823, 0.06660417467355728, 1.7806642055511475, 0.6574647426605225, 0.12142933160066605, -1.3110365867614746, -0.1987607330083847, 0.3602055311203003, 1.6028014421463013, -0.11204436421394348, 0.07947886735200882, -1.6405718326568604, -0.7779995203018188, -0.32296356558799744, -1.0691092014312744, -1.0689998865127563, -0.7230612635612488, -0.746496856212616, 0.2073492556810379, 0.11255908012390137, -0.2219172567129135, 0.008926980197429657, 0.0856335312128067, 1.43911874294281, -0.5613391399383545, -0.9574748873710632, -2.386255979537964, -0.07563021779060364, -1.836759090423584, 0.2986532747745514, -0.2599845230579376, 0.6731259822845459, -0.9262749552726746, -0.2435123771429062, 0.48407140374183655, -1.4685194492340088, 0.6003801822662354, 0.9118181467056274, 0.0724729374051094, -0.30633798241615295, -1.6365638971328735, 0.1826886087656021, 1.451716423034668, -0.8507113456726074], [-0.2785350978374481, -0.6629085540771484, -0.8724465370178223, -2.752094268798828, -0.41119346022605896, -1.3856855630874634, -1.0512104034423828, -0.7097634077072144, -0.8636031150817871, -0.1578086018562317, -1.7641279697418213, -0.9248174428939819, -1.4972546100616455, -0.15537725389003754, -2.465202569961548, 0.8156102299690247, 0.0700269564986229, 0.36098533868789673, -0.142648845911026, -1.011237382888794, 0.9513832330703735, -0.5256244540214539, 1.1051578521728516, 0.22070758044719696, -0.0742807388305664, -0.5022944808006287, 0.10707352310419083, 0.2401752471923828, 1.3606103658676147, -0.6580392122268677, -1.3586623668670654, -1.2089110612869263, -0.5620055198669434, 1.3940876722335815, 0.17345976829528809, 0.8481024503707886, -0.4962729811668396, -1.3774605989456177, -1.2617310285568237, 0.13828426599502563, 0.11431663483381271, -0.7444740533828735, -0.15408408641815186, -0.1681920886039734, 0.6890707015991211, 1.6059447526931763, 0.9953410029411316, 0.37383416295051575, 0.9203439354896545, -2.8677761554718018, 0.7432404160499573, -0.24079464375972748, 0.3463539481163025, 0.5564854145050049, 0.882723867893219, 0.8887633085250854, 1.987752914428711, -0.08600812405347824, 0.9350391626358032, 0.9620856642723083, 0.2481653243303299, -0.5735819935798645, -2.760925531387329, -1.5148344039916992], [0.011232666671276093, -0.25850504636764526, -0.7061887979507446, -0.17139632999897003, -0.4205503463745117, -1.506915807723999, -0.5479557514190674, 0.8798266649246216, -0.8083734512329102, -0.33601030707359314, -0.3916877508163452, -0.7703686356544495, 1.1039572954177856, -1.3068733215332031, 2.095057487487793, 0.1433403491973877, -0.9241611361503601, -0.062284916639328, -2.3890414237976074, 0.5042956471443176, 0.0437421016395092, -0.23194627463817596, -0.5615783929824829, -0.21668922901153564, 0.15444467961788177, -0.3671530485153198, -0.0031380602158606052, 0.33179229497909546, 0.93886798620224, -0.7696710824966431, -0.35869601368904114, -0.09828030318021774, 0.8767465949058533, 0.22171130776405334, -0.046109698712825775, 0.5611940622329712, 2.4397740364074707, 1.0238010883331299, 1.4334332942962646, 0.11438291519880295, 0.31387996673583984, 0.5312432646751404, 0.8300175666809082, 0.7997276186943054, -0.23476144671440125, -0.8057528734207153, 1.2208030223846436, 1.0425117015838623, 1.6430110931396484, 0.9987555146217346, 1.2675387859344482, 1.3641273975372314, 0.7015225887298584, 1.7459698915481567, -0.43096598982810974, 0.8127769827842712, -1.850000262260437, 0.7834790945053101, 1.413954734802246, 0.24648086726665497, 0.23948819935321808, -1.7027994394302368, 0.34764280915260315, -0.6209062337875366], [-1.5273724794387817, 1.160088300704956, -0.6445695161819458, 0.11328184604644775, 0.12818816304206848, 0.7480242848396301, 0.1944654881954193, 0.8544360399246216, -0.9862487316131592, 0.2106913924217224, -0.3613921105861664, 0.8915079832077026, 1.2176328897476196, -1.3948410749435425, 0.8448638916015625, 1.2087934017181396, -0.34236520528793335, -1.637668251991272, -0.1767025887966156, 0.5441097617149353, -1.8411355018615723, 1.5344318151474, -0.33100417256355286, -0.43146345019340515, -0.8191317319869995, 0.4373684525489807, 0.4916088581085205, 1.8185468912124634, -0.9655119776725769, -0.9112865328788757, 1.3858939409255981, -1.626473069190979, 0.14321595430374146, 0.7565456032752991, 1.9630175828933716, 0.26489391922950745, -1.8316935300827026, 0.7710614204406738, -1.2686614990234375, -0.914836585521698, 0.8724901080131531, -1.4285824298858643, -1.9026280641555786, -0.018578078597784042, 0.12840919196605682, -0.6151548624038696, 1.3669395446777344, 1.5859416723251343, 0.21835067868232727, -2.3057031631469727, 0.8243124485015869, -1.027009129524231, 0.56856769323349, 1.129075288772583, 0.6975172162055969, 0.7039013504981995, 0.6679455041885376, 1.1936262845993042, 0.097117580473423, 0.3288562595844269, 2.3844785690307617, -0.45581209659576416, 0.9395917654037476, 0.6643127799034119], [-0.8787081241607666, -0.1571906954050064, 0.6062591075897217, -0.5971665382385254, -0.1426994353532791, 1.1777942180633545, -0.13956224918365479, -0.6153873801231384, 0.39043232798576355, -1.348960041999817, -0.5778849124908447, 0.7328833341598511, -0.8063214421272278, -0.566362202167511, -0.6657438278198242, 0.17088152468204498, 0.3603273034095764, -0.5083157420158386, -1.0486860275268555, 0.22020749747753143, 0.5250718593597412, 1.2168083190917969, -2.188455104827881, 0.38334953784942627, -1.2036278247833252, 0.18021443486213684, 0.08545705676078796, -0.09042251110076904, 0.09340774267911911, 0.622756838798523, 1.7069731950759888, -0.017170380800962448, 0.12326889485120773, 0.24783937633037567, 1.477905035018921, 0.8158515095710754, 0.9373174905776978, -1.0723943710327148, 0.9090606570243835, 1.086333155632019, -2.5968692302703857, 0.12523554265499115, 1.6366245746612549, -1.1227576732635498, 0.2994610369205475, -1.256787896156311, -0.9349068403244019, -0.3873152434825897, -0.18204957246780396, 1.4979069232940674, 0.9815046191215515, -0.3556831181049347, 0.7464381456375122, 1.110558271408081, -0.9280557632446289, -0.935370147228241, 0.9176324009895325, -1.0926291942596436, 0.1803746521472931, -0.19914621114730835, -1.1674858331680298, -0.9046652317047119, 0.7658710479736328, 0.7420419454574585], [0.07219559699296951, -2.6983182430267334, 1.0497835874557495, -1.20356285572052, -0.5662943124771118, -0.31807219982147217, -0.5470841526985168, -0.3825349807739258, 0.26097747683525085, 0.5592250823974609, 1.371881127357483, -0.7215520143508911, -1.5444960594177246, 0.46452566981315613, 0.5085843205451965, -0.02114829607307911, -0.4700925350189209, -0.9786322116851807, 1.9962708950042725, -0.42344728112220764, -0.013571429997682571, -0.3305831253528595, 1.6993962526321411, 0.3606944978237152, 0.24226269125938416, 0.5780928730964661, 0.1112702488899231, 0.44046255946159363, 2.064565658569336, 0.42524680495262146, -0.8450504541397095, -0.9903991222381592, 0.9757235050201416, -1.285179615020752, -0.9773176908493042, -0.08429291099309921, 0.7434308528900146, 1.2698805332183838, -1.0369395017623901, 2.030529260635376, 0.9694122672080994, 1.298007845878601, -1.9983890056610107, 0.17352482676506042, 0.23114871978759766, -1.9005179405212402, 1.1226710081100464, 0.2267461121082306, 0.14247430860996246, 1.0827150344848633, 0.4355527460575104, -0.28490298986434937, -0.8057899475097656, 1.9751936197280884, 0.8486906886100769, 0.4543701708316803, -0.8993929028511047, -1.1757259368896484, -0.2175600528717041, 0.4787861108779907, 1.584481120109558, -2.4575350284576416, -1.220913052558899, 0.08391083031892776], [-1.4504941701889038, 0.7982997298240662, -1.2292600870132446, 0.8400881886482239, -0.6267572641372681, -0.32115256786346436, 1.5407899618148804, -1.1434255838394165, -0.5021167397499084, -0.2947221100330353, -2.1523330211639404, 1.3357559442520142, 1.2148383855819702, -0.7155010104179382, 0.5265507698059082, -1.0299873352050781, -1.1763391494750977, -0.5885003209114075, 0.37903544306755066, 0.3050035536289215, -0.4254080355167389, 0.5198630094528198, -0.916467547416687, 0.8142907619476318, 0.36123907566070557, -1.2475988864898682, -0.8452714681625366, -1.106709361076355, 1.1494941711425781, -0.645864725112915, 1.4842902421951294, -1.6071628332138062, -0.5612309575080872, 0.23579898476600647, 1.481021523475647, 0.8151212930679321, -0.6125288605690002, 0.0009448296623304486, 0.280535489320755, -1.4013720750808716, -0.7890299558639526, 1.4150036573410034, -1.4543925523757935, -1.137930154800415, 1.0657976865768433, 0.1617342233657837, 0.3568664491176605, 0.10968279093503952, -1.180159568786621, -0.0880359560251236, -0.3690646290779114, -0.6381690502166748, 0.6983255743980408, 0.7817716002464294, 0.150867760181427, 0.8487433195114136, -0.6189277768135071, -0.9975383877754211, 2.0377583503723145, 0.10755627602338791, -1.4168674945831299, -0.6082882285118103, 1.160494327545166, -0.46250826120376587], [0.5207317471504211, -0.47404831647872925, 1.2614854574203491, -2.1782753467559814, -2.482283353805542, 0.2500201165676117, -0.17065958678722382, 0.24911655485630035, -1.3788615465164185, 1.2603635787963867, -1.1761113405227661, -0.42528191208839417, -0.08564697951078415, -0.5769706964492798, -2.307600498199463, -0.9446133971214294, -1.1377331018447876, -1.6258375644683838, -0.6751319766044617, -0.05258616432547569, 1.4269319772720337, 1.5311086177825928, -1.5546448230743408, -0.05194956436753273, -0.5305886268615723, 0.4358050525188446, 1.4590305089950562, -0.1461292803287506, 0.265985906124115, -0.6227387189865112, -1.3318336009979248, -0.6935409903526306, -0.42315250635147095, 0.5920334458351135, 1.1580619812011719, -0.6073222160339355, -0.2215316742658615, 1.5418384075164795, 0.26566046476364136, -0.403560996055603, -2.297161340713501, -0.08645657449960709, 0.1650792807340622, -0.7444511651992798, 1.9723305702209473, 0.07881627231836319, 0.017076406627893448, 0.9357033967971802, -0.9982542395591736, 0.9422317743301392, -0.3118239939212799, 0.5561637878417969, 1.6017980575561523, -0.10106205195188522, 0.207895889878273, -0.2803812325000763, -1.7241662740707397, 0.2332981675863266, -0.10561864078044891, -0.44286423921585083, 0.18891334533691406, -0.3710887134075165, 1.201319932937622, -0.5088945627212524], [-0.2656465172767639, -0.8710291385650635, 2.1868879795074463, 1.636224627494812, 0.14605171978473663, 2.1146953105926514, -0.7631312012672424, -1.1170339584350586, 0.46141380071640015, -1.0110872983932495, 0.1621343195438385, -1.1808428764343262, -0.628970205783844, 0.7073673605918884, -0.40030044317245483, 0.3679584264755249, 0.6567790508270264, 0.3072836101055145, -1.1301623582839966, 0.6951648592948914, -0.8183199763298035, -0.5939211249351501, -1.1993745565414429, -0.08687107264995575, -0.0061626429669559, 0.7631582021713257, -0.09050136059522629, -0.9222702383995056, -0.22096918523311615, -0.6087466478347778, -0.9201270341873169, -0.006659433711320162, 1.9176117181777954, -0.0115074273198843, -0.5094239115715027, 0.8308228850364685, -1.406150460243225, -0.3552456200122833, -0.09612663835287094, 0.8981651067733765, 0.9993375539779663, -0.023872872814536095, 0.9950899481773376, 0.9239875674247742, 1.0602725744247437, -0.9701498746871948, -0.7795233130455017, -0.0009082496981136501, -1.0282962322235107, 0.19927425682544708, 0.734313428401947, -0.4002256393432617, -2.47255539894104, -0.43809494376182556, 1.9890178442001343, 0.47148993611335754, 1.504378318786621, 1.6361844539642334, 0.05382627993822098, -1.2263704538345337, 0.5393694639205933, -0.41458579897880554, -0.12950220704078674, 0.12824657559394836], [0.3338536322116852, -0.13330340385437012, 0.7470500469207764, 0.11369548738002777, 1.281502604484558, 0.1464141458272934, -0.1853628009557724, -1.3464630842208862, 0.10430309176445007, -1.198045015335083, -1.614116907119751, -1.0639615058898926, 1.7971880435943604, -0.4200412631034851, 0.9078224897384644, -0.5936419367790222, -0.10671678185462952, 0.6359067559242249, -1.1091161966323853, 1.4116162061691284, 0.026806941255927086, -0.5072875618934631, 1.086488127708435, -0.9946612119674683, -0.30334585905075073, -0.04735065624117851, -0.7303575277328491, -0.28250396251678467, -1.763830304145813, -0.22170396149158478, -0.5626192092895508, 0.3031880557537079, 1.4639239311218262, 0.009029016830027103, -1.199134349822998, 0.7669590711593628, 0.7267561554908752, 1.2286112308502197, -0.8983666300773621, -1.7425743341445923, 1.6125006675720215, 0.7119104862213135, -1.544838309288025, -0.37743037939071655, -0.5717669725418091, 0.8504497408866882, 0.5513451099395752, -0.5523519515991211, 0.7243079543113708, 1.4237490892410278, 1.2104593515396118, -1.275492787361145, 0.013819165527820587, 0.2965468764305115, -0.751895010471344, 0.8462135195732117, -1.1785094738006592, -0.6252390742301941, 0.2862691581249237, -0.4043824076652527, -0.5463088154792786, -0.757667064666748, -1.8426728248596191, -0.5019940137863159], [-0.8990193009376526, -0.7525035738945007, 1.1474056243896484, 1.691196322441101, 1.9548331499099731, 0.5089902281761169, 1.595553994178772, -0.36899563670158386, -1.4725834131240845, -1.695886254310608, 0.31744107604026794, -0.9130204319953918, 0.8772814869880676, 0.304727166891098, -0.8960328102111816, 0.501782238483429, -0.36367955803871155, 0.772500216960907, 0.7829579710960388, 1.4844003915786743, -0.7916111946105957, -1.2072886228561401, -0.9335635900497437, -0.39640602469444275, 1.9689357280731201, -0.8502616286277771, 0.604335606098175, 0.7084406018257141, 0.16683213412761688, -0.9774788022041321, 0.1645854264497757, 0.4764727056026459, 1.167218804359436, 0.46992185711860657, 0.4741460978984833, 0.8907458782196045, 0.8670979142189026, -0.4730142652988434, 0.37989142537117004, -0.24972236156463623, -0.7186911106109619, 1.3296819925308228, -0.9356851577758789, -0.7330659627914429, 0.20234236121177673, -0.8884673118591309, -0.24445763230323792, 0.04222693666815758, 1.0369462966918945, -0.0253333430737257, 0.5606533288955688, -0.24620437622070312, 1.7530359029769897, -1.6774168014526367, 0.31367436051368713, 1.3506510257720947, -2.2852468490600586, 0.13334114849567413, 1.231713056564331, 1.532043218612671, -0.4125578701496124, -2.044110059738159, 0.5118287205696106, 0.5458963513374329], [-0.8659205436706543, -0.4785790741443634, -0.006107627414166927, 1.3924806118011475, 1.7189007997512817, -0.5032863616943359, 1.0467175245285034, -0.7788987755775452, 0.12762288749217987, 0.5897097587585449, -0.08160954713821411, -1.1952506303787231, 0.5491636991500854, -2.0393197536468506, 0.8643437027931213, 0.1784401386976242, 1.0428075790405273, -1.769028902053833, -0.578663170337677, -2.0520503520965576, 1.256005883216858, -0.8349555134773254, 0.9625484943389893, -1.3719308376312256, 0.1431114375591278, -2.3301596641540527, 0.6215775609016418, -0.9188958406448364, -1.8603512048721313, 0.6123930215835571, 0.23051483929157257, 1.547560453414917, 1.0051246881484985, 0.2633139491081238, 1.7696149349212646, -0.4343639314174652, 1.8644111156463623, 0.1951695680618286, 1.9947956800460815, 0.01700594648718834, 0.20988143980503082, -0.3639288544654846, 0.6196889281272888, -0.7902328372001648, -0.04653298482298851, -1.7382307052612305, -1.4503273963928223, 1.3433600664138794, -0.2692100405693054, 0.28348663449287415, 1.9778311252593994, -0.016993463039398193, 0.27614790201187134, 0.6825428009033203, 1.04738187789917, -0.8664911389350891, 1.63186514377594, -0.2153470665216446, -0.26692646741867065, 0.8524577021598816, -2.2169017791748047, -0.586979866027832, 2.2375125885009766, 1.1300767660140991], [-0.3027249872684479, 0.6696269512176514, 0.6815677285194397, 0.6766372919082642, 1.6869728565216064, -0.005599954631179571, 0.6967061161994934, -0.3175573945045471, 0.12793205678462982, -0.14335140585899353, 1.4212863445281982, -1.1871871948242188, 1.0820127725601196, 1.1215332746505737, -0.9721742868423462, -1.4831559658050537, -0.7669476866722107, 1.4643114805221558, 2.111870765686035, -0.17905080318450928, -0.7103469967842102, -0.2102755755186081, 0.9167158603668213, -0.2932855188846588, -0.6959008574485779, 0.535785973072052, 0.5230274796485901, 0.8104548454284668, -1.3146101236343384, -1.4333168268203735, 0.8376737833023071, -1.683534026145935, 0.08723925799131393, 0.5500550270080566, -2.1669106483459473, 1.1798101663589478, 1.4588764905929565, 0.34879225492477417, 0.5495902895927429, 0.5972134470939636, 0.43488532304763794, -0.055273108184337616, -0.4979236423969269, -0.07385403662919998, 0.5475868582725525, -0.05781735107302666, -0.8801138401031494, 0.08267766982316971, 0.6281949877738953, -0.17727883160114288, -0.05769480764865875, -0.18039512634277344, -0.9782109260559082, 0.44737371802330017, 0.998969316482544, -0.8654561042785645, 0.7153110504150391, -0.5545690059661865, -0.8243669271469116, 0.2688395380973816, -0.11198300868272781, 0.2610182464122772, 0.3045765459537506, 1.0066601037979126], [-1.8985393047332764, 1.4993054866790771, -0.8151028156280518, -0.00043487572111189365, 1.2774357795715332, -0.7080203294754028, -0.03173613175749779, -0.4948962330818176, 0.31602197885513306, -1.0085678100585938, 0.42358899116516113, -0.8369118571281433, -0.9303677678108215, -1.1349005699157715, -1.5007256269454956, -0.39351779222488403, -1.194884181022644, -0.0044914246536791325, -0.10188344120979309, 0.554245114326477, -1.9885340929031372, 0.07829614728689194, -1.1468873023986816, -0.7569783926010132, -0.0687042847275734, 2.562908887863159, 0.5319324135780334, -0.3483223021030426, -0.3288261592388153, -0.6169410347938538, 0.8611377477645874, 0.7276912331581116, -1.3320000171661377, -0.6966923475265503, -0.8635414242744446, 0.2025114893913269, -1.4398279190063477, -0.441474586725235, 0.6510716676712036, -1.3319957256317139, -0.7987509369850159, 0.47845739126205444, 0.7929760813713074, 0.14229150116443634, -1.450157880783081, 0.4267800450325012, 1.296471357345581, -2.2682409286499023, -0.17856672406196594, 2.453338146209717, 1.3134068250656128, -0.740034818649292, 0.9203216433525085, 0.11266758292913437, -0.3731781244277954, -0.3772324323654175, -1.2404204607009888, 1.239588975906372, -0.6484809517860413, 0.3378134071826935, -0.8430063128471375, 0.791066586971283, 0.647395670413971, -1.231092929840088], [-1.1588897705078125, -0.6319310665130615, -0.9838714003562927, 0.6269710659980774, -0.4787389039993286, -1.447049856185913, 1.7119784355163574, 1.9781363010406494, -0.8789501190185547, 2.294905424118042, -0.29518288373947144, -0.46072009205818176, -0.22756294906139374, 0.8215656280517578, 1.0499223470687866, -1.667711615562439, -1.9399194717407227, -1.7014265060424805, 0.4147917628288269, 1.5146205425262451, 0.20073433220386505, -1.2021267414093018, -1.122869849205017, 0.41787561774253845, 1.1273261308670044, 0.3214944005012512, 0.07380228489637375, 1.9256097078323364, -1.0091369152069092, -1.4339851140975952, 0.9733384251594543, -1.2795374393463135, -1.043871283531189, -1.6255078315734863, -0.8327322602272034, -1.3575505018234253, -1.647412896156311, -0.4849248230457306, -0.02176700532436371, 1.9647367000579834, 0.1561572253704071, 0.5321208238601685, 0.9024994969367981, 0.8832643628120422, -0.6166841387748718, 0.6831920146942139, 1.6023211479187012, -1.0266045331954956, -0.6476632356643677, 0.9210080504417419, 0.41472405195236206, 0.7626516819000244, 0.15105319023132324, 1.0737719535827637, 2.4709818363189697, -0.033322181552648544, 1.059937834739685, -0.08875317126512527, 0.11690216511487961, -1.2948529720306396, 0.24074849486351013, 1.382431149482727, 1.1268140077590942, 1.7216527462005615], [0.2855171859264374, 0.1014852449297905, 1.9055020809173584, 1.025126338005066, 1.2929686307907104, -0.1525413691997528, -1.6388942003250122, -2.266129493713379, -0.23367834091186523, 0.42103007435798645, -0.29763320088386536, 0.47019460797309875, 1.0990419387817383, 1.249765157699585, -2.5951344966888428, -0.572123110294342, 1.307037115097046, 0.6728941202163696, 0.4609864056110382, 0.5477870106697083, 0.07579132169485092, 1.7322072982788086, -0.33962541818618774, -0.13524693250656128, 0.004615729674696922, 0.04845666512846947, -1.7068791389465332, 0.28175604343414307, 1.2507323026657104, -0.5993062853813171, -1.8356435298919678, 0.6443067789077759, 0.47882699966430664, -1.4163438081741333, 0.3659631013870239, -1.0453574657440186, 0.08215891569852829, -0.47010964155197144, -0.5220443606376648, -0.25497719645500183, 1.4332691431045532, 0.6663991808891296, 0.19938986003398895, -0.7092946767807007, 0.7495630979537964, 0.7209610939025879, -0.6333151459693909, -1.7786891460418701, 0.4781607687473297, -0.7901928424835205, -0.9641073942184448, -1.981986403465271, 0.383926123380661, 0.5863216519355774, -1.236867904663086, 1.7386608123779297, 0.5825101733207703, -1.5107629299163818, -0.059729382395744324, -0.37142181396484375, 0.7477549314498901, 0.33963048458099365, -1.0834643840789795, -1.1263099908828735], [0.025411546230316162, 0.6930753588676453, -1.1990623474121094, 0.0907495766878128, -0.08196116983890533, -1.0269750356674194, 0.29286321997642517, -0.9024373292922974, 0.12598149478435516, -0.660661518573761, 0.2015279233455658, 2.214141368865967, 0.8849280476570129, -0.23028720915317535, 0.890465259552002, -0.16990415751934052, 0.3851449489593506, -1.2366275787353516, -1.1119890213012695, 1.4890145063400269, -0.5945765972137451, -1.2480508089065552, -2.291332960128784, -1.5419979095458984, -0.20293505489826202, 0.3116743862628937, 1.3041781187057495, -0.7692801356315613, 0.24820531904697418, -0.30700239539146423, 0.7260138392448425, 1.0544391870498657, 0.46289682388305664, -0.766150951385498, -1.4445946216583252, -1.3551580905914307, 1.3259758949279785, -0.16413965821266174, 0.0763433650135994, -0.5748955607414246, 0.44460049271583557, -0.3768846094608307, -1.2863361835479736, -1.3844728469848633, -1.61056387424469, -0.7521074414253235, 0.10722339153289795, 1.1021171808242798, -0.9507332444190979, -1.2450597286224365, -0.6924980878829956, 0.9484909176826477, -0.12649022042751312, 0.07590442150831223, -0.337575763463974, 2.486396074295044, -2.0073933601379395, 0.5466623902320862, 1.6923670768737793, 0.7702809572219849, 1.657354712486267, 0.35200121998786926, 0.5534995198249817, -0.9515728950500488], [-0.3948342502117157, 1.0008161067962646, -2.168935537338257, 0.692918062210083, -1.6108760833740234, -1.1890349388122559, 0.23860310018062592, -1.743600845336914, 0.011336330324411392, 1.1791350841522217, 0.8443763852119446, -0.5750879645347595, 0.11544305086135864, 0.868604302406311, -0.7045291662216187, -0.2878710925579071, -0.9706820249557495, 1.1209619045257568, -0.2108241468667984, 0.8220179677009583, 0.3203924000263214, -1.0299862623214722, 1.3902682065963745, 0.6377459764480591, 0.5646610856056213, -0.8683424592018127, 0.3855702579021454, 0.9561036229133606, -1.5585750341415405, -2.0754759311676025, 0.34907588362693787, -0.13963209092617035, -1.7227745056152344, 0.1543148308992386, -0.8337682485580444, 1.1175267696380615, 0.15370461344718933, 1.13191819190979, 0.8449441194534302, -1.005569338798523, -0.18147757649421692, 0.16161571443080902, 1.175668478012085, 1.1166139841079712, 1.0627390146255493, 0.45657962560653687, 1.1768242120742798, -1.3738497495651245, -0.36389368772506714, -1.0474035739898682, 0.08506567031145096, -0.43754705786705017, 0.49507665634155273, 0.3404097557067871, -0.44221630692481995, 0.26063841581344604, -0.7082919478416443, 1.448610544204712, 0.5562129020690918, -0.9814912676811218, -0.03937581554055214, 0.3822537958621979, -0.5686625838279724, -1.0448404550552368], [0.3203706741333008, 0.7206482887268066, 1.6128215789794922, 0.09527681767940521, -0.1527377814054489, -1.3657947778701782, -0.7720710635185242, 0.8513017892837524, 0.8225743174552917, 0.9847832918167114, 1.634112000465393, 2.200892210006714, -0.8245077729225159, 1.1863833665847778, -1.6139650344848633, 0.2995767295360565, -0.9489049315452576, -1.5754048824310303, -0.07640478014945984, 1.4715173244476318, 1.2606325149536133, 0.24954183399677277, 0.0663161352276802, 0.37387245893478394, 0.9401170015335083, 0.07315853983163834, -1.3270649909973145, -0.6770341396331787, 0.8496944904327393, -2.3081881999969482, -0.6940298676490784, 0.5113298892974854, -0.2145959585905075, 0.6123887896537781, 0.9096362590789795, 0.27384936809539795, -0.8643849492073059, -0.047252848744392395, -1.3816850185394287, -0.8430108428001404, 0.9931532740592957, -1.5844898223876953, 1.266549825668335, -0.5738623738288879, 0.8574046492576599, 1.836601734161377, -1.4682693481445312, 1.3753032684326172, -0.026339629665017128, 1.2969316244125366, 0.8865423202514648, 0.0894538089632988, 1.5943702459335327, -1.0962880849838257, 0.6159204244613647, 0.9408054947853088, 1.2700254917144775, -0.2266572266817093, -0.859504222869873, 0.32910463213920593, 1.6260699033737183, 1.6738163232803345, -0.5009697079658508, 0.7685824632644653], [2.1910531520843506, -0.30458810925483704, -0.7917880415916443, -1.732547640800476, 1.0097887516021729, 0.5973383784294128, -0.1359463930130005, -1.0654844045639038, -2.192194700241089, -0.5920485854148865, -0.6134740710258484, 0.6102712750434875, -1.018630027770996, 1.1120229959487915, -0.016555802896618843, 0.6939912438392639, -0.7020915746688843, 0.06853816658258438, 1.1203542947769165, 1.838990330696106, 0.9944417476654053, 0.5713159441947937, 1.6780414581298828, 1.0382057428359985, 0.5449368357658386, 0.29810190200805664, 0.16044554114341736, -0.6018103361129761, 1.04896879196167, 0.5288630127906799, -1.6382266283035278, 1.42683744430542, 0.08719362318515778, -0.22507794201374054, 2.2252748012542725, -1.1171386241912842, 0.930846631526947, 1.1155014038085938, 0.5957512259483337, -1.738932728767395, 0.287984699010849, 0.8418467044830322, 2.490412950515747, -1.7646769285202026, 2.946180582046509, 0.1624966710805893, -1.0947448015213013, 1.0672619342803955, 0.08590659499168396, 0.6592307686805725, -0.053255919367074966, 0.3476142883300781, -0.7480654716491699, -2.072587728500366, 0.8100593686103821, -1.0214388370513916, -0.6235158443450928, 0.4305790066719055, 1.2152539491653442, 0.014892974868416786, 3.4258275032043457, -1.4899773597717285, 0.9566592574119568, -0.49837639927864075], [0.6371220946311951, -1.3208279609680176, 1.8408626317977905, -0.40121421217918396, -0.9717197418212891, -0.35028794407844543, 0.2035052478313446, 0.5120530128479004, -0.399944543838501, -1.3461763858795166, -0.7894006371498108, 0.04649687558412552, -0.7459391951560974, 0.4394569993019104, 0.32933205366134644, 0.5390468835830688, 1.8097928762435913, 0.00997494999319315, -0.605309784412384, 0.33682796359062195, -0.2764519453048706, -0.29623720049858093, 0.05099571868777275, 0.7626768946647644, 0.5057122707366943, -0.2918796241283417, -1.4646594524383545, -0.2392304390668869, 0.27454325556755066, 0.6228905320167542, -1.499869465827942, -0.12139499187469482, -0.26156237721443176, -1.3676791191101074, -0.22674250602722168, 0.06153028458356857, 0.7533835768699646, 0.7058109045028687, -0.5511220097541809, 0.5420278310775757, -0.8356747031211853, -0.4689192473888397, 0.5644643306732178, 2.4241726398468018, -1.0688190460205078, -0.13291426002979279, -1.5608110427856445, 1.225233793258667, -0.07762423157691956, 1.3227508068084717, 0.06212301552295685, 0.8553243279457092, -0.7534149289131165, -0.4110206961631775, -1.0515778064727783, 0.3560187816619873, 0.5709007978439331, -0.7729389071464539, 0.7279743552207947, 0.11216127127408981, -0.41928228735923767, 1.1644365787506104, -1.1192383766174316, 2.11112904548645], [-0.4048338532447815, 0.2353382408618927, 1.060062050819397, 0.9294002056121826, -0.7357490658760071, 0.8367615938186646, 0.029310159385204315, 0.5285176634788513, 0.29871419072151184, -1.3283584117889404, -0.18491892516613007, -0.6030037999153137, 0.17042021453380585, 0.04973555728793144, -1.6388901472091675, 0.6579630970954895, 0.47145727276802063, -0.48085013031959534, 1.2385376691818237, -0.5386250019073486, 0.9098982810974121, -0.16753444075584412, -0.8337604403495789, -0.060134224593639374, 0.2527600824832916, 1.2131272554397583, -0.2888283431529999, 0.1734461784362793, 0.16338779032230377, 0.2534649074077606, 0.8805603981018066, 1.3064383268356323, 2.1032235622406006, 1.3180489540100098, 0.1534062922000885, 1.766371488571167, -2.3249783515930176, 1.7252775430679321, 0.33675700426101685, -0.5257902145385742, -1.6366522312164307, 1.7578378915786743, -0.14872190356254578, -0.038191668689250946, 0.8483803868293762, -0.7686263918876648, 0.2297976315021515, 0.28936153650283813, 0.0775625929236412, 0.5156552195549011, -0.13378627598285675, 1.0265777111053467, -0.829371988773346, -0.5978162288665771, -2.101614475250244, 0.6603655815124512, 1.1326020956039429, 0.5730634331703186, 0.7613288164138794, 1.8826249837875366, 1.1683276891708374, 0.04017152264714241, 2.3122055530548096, 0.2342269867658615], [-0.8089086413383484, -0.7131152153015137, 1.0122586488723755, -1.3199330568313599, 0.32436487078666687, -1.370063066482544, 0.842069149017334, 0.29049691557884216, 0.4335176646709442, 1.2220942974090576, -0.11805938929319382, -0.4130179286003113, -1.1407930850982666, 0.5478845834732056, -0.5773306488990784, 0.7010765671730042, -1.5627434253692627, 0.9455586075782776, -0.4027591347694397, 0.9104289412498474, -0.33637773990631104, 0.2559921145439148, 0.8072906136512756, 0.013038840144872665, -0.8124627470970154, -0.39893820881843567, 0.657331645488739, 1.276366949081421, -1.2587813138961792, 1.7553701400756836, 0.9780192375183105, -1.1667877435684204, 0.6633638739585876, -1.1118580102920532, 0.22735542058944702, 1.2387161254882812, -1.0268203020095825, 1.0835785865783691, -0.355430006980896, 0.3543192148208618, -0.10320068150758743, 0.7340570688247681, -1.0929182767868042, 1.1885188817977905, 2.0105338096618652, 1.2147786617279053, -0.34240981936454773, 0.30532601475715637, 1.3061708211898804, 1.259372353553772, 3.512538194656372, 2.350494146347046, -1.0993211269378662, 1.21730375289917, 0.38352370262145996, -0.10476729273796082, -0.7096776962280273, -1.0016603469848633, 2.5784521102905273, -1.2141518592834473, 1.0508586168289185, -2.357794761657715, -0.4423932135105133, 0.5835533738136292], [-0.6859759092330933, -0.6079280376434326, 0.22303296625614166, 2.235941171646118, 0.535258412361145, 0.1903761923313141, 0.17011277377605438, 1.17267644405365, -0.624803900718689, -0.8762174844741821, -0.7788495421409607, -0.9221518039703369, 0.3098281919956207, 0.19886086881160736, 0.8318952322006226, 0.6271485686302185, 0.26280614733695984, 0.4422403872013092, 0.5790965557098389, -0.6542630791664124, 0.46667468547821045, -0.03574617952108383, -0.294234961271286, -0.43844249844551086, -1.3855631351470947, 0.6392174959182739, -0.8763952255249023, -0.3261869549751282, -0.9637042284011841, -0.06404118984937668, -0.5819639563560486, -2.4144017696380615, 0.6189284920692444, 1.9526188373565674, -1.4843426942825317, 0.749783992767334, -0.6785884499549866, 1.4346389770507812, 1.2684967517852783, -0.31271347403526306, -0.2840356230735779, -0.018815388903021812, 0.21660766005516052, -1.2323590517044067, -0.24802882969379425, 0.005714881233870983, 0.9966167211532593, 0.868768572807312, -1.8498984575271606, -1.8714940547943115, -1.065344214439392, -0.2297104299068451, -0.5070009827613831, 0.44593095779418945, -1.3988735675811768, -1.0560609102249146, -0.5477825999259949, 0.6070548892021179, -0.029053661972284317, -0.004888685420155525, 2.4370205402374268, 0.9471511840820312, 1.7090528011322021, -0.5408009886741638], [1.4324665069580078, -1.261770486831665, 0.46448710560798645, 0.07375481724739075, -0.07005281001329422, -0.17322774231433868, -1.3664277791976929, -0.10005649924278259, -0.11968402564525604, -2.579829216003418, 0.4816431701183319, -0.842943549156189, -1.716072916984558, -0.26510316133499146, 0.5487895607948303, 0.2853875458240509, 0.9586259126663208, 0.9730822443962097, 0.33551523089408875, -0.6083105206489563, -0.22602836787700653, 0.6807120442390442, -1.2040235996246338, -0.5469620227813721, -1.4024049043655396, -0.4172769784927368, -0.9770686626434326, 0.4414920210838318, 1.7271637916564941, 0.017658809199929237, 2.1813793182373047, 1.3322967290878296, -1.132429599761963, 0.4967944622039795, 0.8621864318847656, -0.7039939165115356, -0.2664047181606293, 0.5479145050048828, -0.41709083318710327, 0.4411662518978119, 0.5753210186958313, -1.3450608253479004, 0.2796633839607239, 0.32346010208129883, -1.279248833656311, 0.2971314787864685, -1.0268875360488892, 0.1448114663362503, 1.0670275688171387, -0.4705415368080139, -1.2912325859069824, -0.8453501462936401, 0.48057815432548523, 0.2947351634502411, -0.360294908285141, 1.3444589376449585, 0.2711009383201599, 1.854138970375061, 0.3053944706916809, 1.7745012044906616, -0.8312087655067444, -0.8315703868865967, 0.36482807993888855, 2.1959009170532227], [0.43148091435432434, 0.9899522066116333, 0.8050577044487, 0.8258443474769592, -1.146270751953125, 0.08913060277700424, 1.0610265731811523, 2.578991651535034, -1.4017186164855957, -0.4577997028827667, -0.4700409770011902, -0.00877299066632986, 0.6376495361328125, 0.18437667191028595, 0.14803417026996613, 0.8061597943305969, -1.7381349802017212, -0.3096756935119629, -0.26024287939071655, 0.12238936126232147, -1.0625507831573486, 0.8321888446807861, -0.02038096822798252, 0.4306897222995758, -0.22803716361522675, 0.7076520919799805, -1.1794812679290771, 0.18658766150474548, 0.7198195457458496, -1.8559165000915527, 1.1697595119476318, -1.7780210971832275, -0.8873181343078613, -0.6059862375259399, -0.42147865891456604, -1.8084533214569092, 0.9036033749580383, -0.1824558973312378, 0.9483776092529297, 1.0277963876724243, -0.9069378972053528, -0.37342560291290283, -1.7744253873825073, 0.4072873890399933, -1.2687137126922607, -0.28815799951553345, 0.07433055341243744, -0.6047185659408569, 0.7059521079063416, -1.7431856393814087, -0.4089462161064148, 0.9632914066314697, 0.16721919178962708, -1.5204448699951172, 0.494003564119339, 1.1428368091583252, 0.7187699675559998, -0.1144285500049591, -0.4792347252368927, 0.7612625956535339, 0.061656009405851364, -1.7889676094055176, 0.249480202794075, 0.4426195025444031], [-1.9996087551116943, -1.5968024730682373, 0.18268340826034546, 0.27443015575408936, -0.8630308508872986, -1.4287697076797485, 0.4795311987400055, -0.6632331609725952, -0.6621701121330261, 0.05232492834329605, -1.6300228834152222, -2.0751101970672607, 1.2137960195541382, -0.8183058500289917, 0.8124080896377563, -1.4878208637237549, -0.19433513283729553, 0.6987113952636719, -0.040656931698322296, -0.6462888717651367, -0.08537061512470245, 0.1874205470085144, 0.5011400580406189, 1.0888793468475342, -0.37589651346206665, 0.8762297034263611, -0.06772047281265259, 0.8225961923599243, 0.41163307428359985, 0.37781110405921936, -1.5653287172317505, 1.289953589439392, 1.3261981010437012, -0.01822558417916298, 0.7088755965232849, -0.0757376104593277, -1.1441298723220825, -0.04401396960020065, 0.6007847785949707, -0.2850187122821808, 1.0726196765899658, 0.2576970160007477, -0.03222518041729927, -0.9442849159240723, 1.2697409391403198, 0.17889289557933807, -0.7233811020851135, -0.735592246055603, -0.6144658923149109, 1.5059106349945068, -0.03168213739991188, 0.40546971559524536, 1.4418504238128662, -1.0996030569076538, -1.7195903062820435, 0.24697594344615936, 0.5514310002326965, 0.43690964579582214, -0.2594466507434845, -0.5478449463844299, 0.38346949219703674, -0.24266311526298523, 1.0809433460235596, 0.2955531179904938], [-0.15097080171108246, 1.6636244058609009, -2.9565765857696533, -0.3030233085155487, -0.02610171213746071, 1.3437315225601196, 1.3950499296188354, -1.4347606897354126, 0.14617076516151428, -0.4200804829597473, 1.153320550918579, 1.5313119888305664, -1.2745575904846191, -0.11173292249441147, 0.7768034934997559, 0.6412850618362427, -1.3520472049713135, -0.467576265335083, -1.802449107170105, 0.6245582699775696, 0.4308500587940216, 1.8404979705810547, -0.405060350894928, 0.05232555419206619, -2.033761978149414, 0.49706992506980896, -0.9297811388969421, -0.22394070029258728, -0.7734843492507935, 1.8597564697265625, 0.7953280210494995, 0.057944003492593765, -2.1555426120758057, -0.06254522502422333, 0.9973005652427673, -1.0400785207748413, 0.49791571497917175, -0.4473029673099518, -0.9937461018562317, -0.28558823466300964, 0.7893282771110535, 0.09025850147008896, -0.2657129764556885, -1.161238670349121, 0.43529924750328064, -1.037365198135376, -0.4828658998012543, 0.033427584916353226, -1.5401790142059326, -1.0677655935287476, 0.9152557253837585, 1.4785875082015991, -0.10023951530456543, -0.3758542239665985, 0.0909341350197792, 0.4019162058830261, 1.016229510307312, 0.8459835648536682, 0.05969676747918129, -1.4335178136825562, -1.209606647491455, -1.6286224126815796, -2.1412136554718018, -0.8449989557266235], [0.22496409714221954, 1.2724449634552002, -0.9581732749938965, -1.0334810018539429, 1.6821857690811157, -1.130943775177002, 0.36263522505760193, -1.204931616783142, 0.8199046850204468, 0.2099161148071289, 0.870332658290863, 0.5609914660453796, 1.0353158712387085, -1.757459044456482, -1.0453276634216309, 0.15878921747207642, -1.4834777116775513, -0.03432357311248779, -1.614335298538208, -0.6643266677856445, -0.13527122139930725, 1.8866130113601685, -0.4206937551498413, 0.5335953831672668, 0.28136155009269714, 0.6770069599151611, 0.4223581552505493, 0.5657811760902405, -1.1472793817520142, 0.1938033252954483, -2.1621224880218506, 0.16695787012577057, 1.1006816625595093, -0.24837486445903778, -0.12627144157886505, 2.007723093032837, -0.1356452852487564, 0.03660570830106735, -0.48207032680511475, -2.356250762939453, -1.6816074848175049, 0.3829978108406067, -1.183808445930481, -1.6103265285491943, -0.5022558569908142, -0.33778464794158936, 1.6155614852905273, 0.7178685069084167, 1.6747305393218994, 1.0612645149230957, -0.5759052634239197, 1.547566294670105, -0.7347884774208069, 0.3533363938331604, -1.7972218990325928, 0.28010547161102295, -0.3478034436702728, -0.749981701374054, -0.23067229986190796, -0.5453359484672546, -1.9467450380325317, -0.27692678570747375, -1.1973321437835693, 0.02702569216489792], [1.2083039283752441, -0.061400581151247025, -0.09181836992502213, -0.21182067692279816, 1.6184526681900024, 0.1987384706735611, -0.6953355073928833, -0.9985535144805908, 0.5081465244293213, 0.10469204932451248, 0.12625539302825928, 0.687401294708252, -0.3236069679260254, 0.9440212249755859, -1.543370008468628, 1.1818947792053223, -2.7076516151428223, 0.09410304576158524, -1.0376592874526978, 1.3649818897247314, 0.571412205696106, -0.6724026799201965, 1.085802435874939, 0.5386379957199097, -0.5609020590782166, -0.8531867265701294, -0.33740419149398804, 0.4835033714771271, -0.7443141341209412, 1.772010326385498, -0.020810367539525032, 0.17221708595752716, -0.6559782028198242, -0.14212757349014282, 0.44681739807128906, -0.03770574927330017, -0.8291603326797485, -0.4728195369243622, -0.256571501493454, 1.9881482124328613, -0.42420437932014465, 0.6041984558105469, -0.5837340950965881, 1.4754095077514648, -0.595061182975769, 1.6695383787155151, 0.7662832140922546, -1.3074451684951782, -0.5752676725387573, -0.9002422094345093, -0.4915635883808136, -0.8387698531150818, -0.5575031638145447, -0.09413358569145203, 0.26229172945022583, -0.8381648063659668, -0.8822680711746216, -0.009503977373242378, 1.6368396282196045, -1.7940465211868286, 1.3145811557769775, 0.3977542221546173, 1.7697070837020874, 1.50128173828125], [0.6402549743652344, 1.2916291952133179, -0.03761259466409683, 0.1551039069890976, 0.14021527767181396, 0.3721298575401306, 0.6450402736663818, -0.6796738505363464, 0.9789353609085083, -0.21920818090438843, -0.897348940372467, -1.807198166847229, -0.23330944776535034, -0.6144269108772278, 0.02656528726220131, 2.3445680141448975, 0.771827220916748, -0.5899155735969543, 1.2703989744186401, -1.1184533834457397, -1.2872146368026733, 0.7282271981239319, -1.0935577154159546, 1.1077436208724976, 0.7455881834030151, -1.034065842628479, -1.1416826248168945, 0.14059598743915558, -1.0692205429077148, 1.6533019542694092, 1.2301627397537231, 1.2840148210525513, 1.2141329050064087, -0.052997689694166183, 0.7472925782203674, 0.8456750512123108, -0.6097276210784912, 0.409013032913208, -0.23081237077713013, 0.9151375889778137, -0.12863284349441528, -0.6964514851570129, -1.5595786571502686, 1.1512938737869263, -0.0030380808748304844, -1.3120083808898926, -0.018879981711506844, -0.7119467854499817, -0.39564207196235657, 0.9655234217643738, 0.48852789402008057, -0.4873271584510803, 0.7079891562461853, 1.2437318563461304, -0.1444372534751892, -1.5876376628875732, 0.5323646664619446, 0.15485528111457825, 0.9409058690071106, 0.3151373863220215, 1.0084391832351685, -1.4428355693817139, 1.9045746326446533, -0.03280366212129593], [-0.9869150519371033, 0.45899614691734314, 0.36175766587257385, 0.9050264358520508, -0.6700659394264221, 1.4564964771270752, -0.5938429236412048, -0.15259987115859985, -0.35587790608406067, 0.9679219126701355, 1.495643973350525, 1.6333563327789307, 0.22789965569972992, -0.4555583894252777, 0.1652538925409317, -0.15966330468654633, 0.7048762440681458, -0.8664847016334534, -0.7579952478408813, 0.025397030636668205, -0.408989280462265, -1.4427627325057983, -0.6920990943908691, 0.18518778681755066, -0.937097430229187, -0.3609588146209717, -0.07131742686033249, 1.6860277652740479, -1.0049166679382324, -0.4186722934246063, -1.681462287902832, -0.3601166605949402, 0.07599496096372604, 0.35071128606796265, -0.9686834216117859, -0.027943449094891548, 1.0598174333572388, -0.17789189517498016, -0.1922026127576828, -0.43289047479629517, -0.9077253937721252, -0.15253911912441254, 1.431931734085083, 0.90525883436203, 1.5276646614074707, -1.3713105916976929, 0.15028372406959534, 0.9269574880599976, 0.1463327407836914, -0.958393931388855, 2.917757749557495, 2.512549638748169, -1.2210454940795898, -0.24821177124977112, 1.2311241626739502, 0.4660934805870056, -2.2579565048217773, -0.2797911763191223, -0.5254288911819458, 0.6034559011459351, -1.365195393562317, -0.6118567585945129, 0.598151445388794, -1.3182905912399292], [-0.5923518538475037, -0.49872466921806335, 0.541968047618866, -0.2741057872772217, 0.43830543756484985, 0.03230785205960274, -1.0850038528442383, -2.3841230869293213, 0.020259369164705276, 0.10827755928039551, -0.19040325284004211, -0.5650038719177246, -0.6388641595840454, -1.9869678020477295, -0.8037092089653015, -0.37249234318733215, -1.3759962320327759, 1.2049604654312134, 0.19912278652191162, -0.2120462954044342, 0.02471109852194786, 0.41104716062545776, 0.9167393445968628, -0.6715790629386902, -0.005451892968267202, 0.7428070902824402, 1.1819065809249878, -0.17593255639076233, 1.5077823400497437, -0.9734882712364197, 1.1446832418441772, -0.10563313961029053, -0.29029566049575806, 0.455375075340271, 1.0187219381332397, -0.6190142631530762, 0.324907124042511, 0.7961666584014893, 0.654895544052124, 1.3439472913742065, -0.7339257001876831, -1.5321842432022095, 0.07331748306751251, -0.5518525838851929, -0.7025889158248901, -0.9544451832771301, 0.593725860118866, -1.1364266872406006, 0.42791688442230225, 0.8067607879638672, -1.6671117544174194, 0.7660567164421082, 1.3797918558120728, -1.5589120388031006, -0.4492292106151581, -0.3959054946899414, 0.7893224954605103, -0.279798686504364, 0.2283134013414383, 0.2677156329154968, 0.3248879313468933, -1.365472435951233, -0.6499380469322205, -0.6712307333946228], [-1.7312204837799072, 4.054174423217773, -0.04703913629055023, -0.8955731987953186, -1.4985681772232056, -1.2355120182037354, -2.48690128326416, 2.0031044483184814, 0.6348209381103516, 1.9207525253295898, -0.3866380751132965, -1.2593973875045776, 1.7198268175125122, -0.8125898838043213, -0.6858231425285339, -0.007332612294703722, 0.22392214834690094, -0.4297085404396057, -0.4025675356388092, 0.27161332964897156, -0.6873979568481445, 0.5681105256080627, -0.08929895609617233, -0.6707029938697815, -0.8893980383872986, -0.7710605263710022, 0.10128717869520187, 1.3714805841445923, 0.36235758662223816, 0.3409368097782135, -0.5901514887809753, 0.1383758932352066, 1.2110575437545776, 1.236687421798706, 0.5578270554542542, -0.12177055329084396, -2.311978816986084, -1.426724910736084, 0.5858139991760254, -1.312401533126831, -3.454063892364502, 0.5256068110466003, 1.0159006118774414, 0.18364040553569794, 2.5709543228149414, 0.8350412249565125, 1.7402409315109253, 0.9988300204277039, -1.760170340538025, -1.1210856437683105, -0.41489988565444946, -0.9953884482383728, -1.1606584787368774, -1.1433069705963135, 1.7756556272506714, 0.3825230300426483, 0.6778146028518677, -0.7648271322250366, 0.750093936920166, 0.4030155837535858, 1.3724408149719238, 0.28395402431488037, -0.04762294888496399, -0.20336630940437317], [-1.6633505821228027, -0.21548114717006683, -0.06333115696907043, 0.3409803509712219, -0.8022773861885071, 1.191968321800232, 0.7944827675819397, 0.7922081351280212, -0.623325765132904, 0.030069757252931595, -0.07307988405227661, 0.4319358468055725, -1.2109041213989258, 0.38133692741394043, -0.06940451264381409, -0.582507312297821, -0.23293541371822357, -0.9990036487579346, 1.2771594524383545, 0.2798304259777069, 0.671004056930542, 0.3987782299518585, -0.8453947305679321, -0.31058400869369507, -0.10903066396713257, -0.5908507108688354, 2.314509630203247, 1.466980218887329, -1.6872332096099854, 2.223937749862671, -1.6781716346740723, -1.2362598180770874, -0.8642367124557495, -0.9836128354072571, 0.1752953678369522, 0.01671784743666649, -0.4376432001590729, 0.4324164390563965, 1.3724970817565918, 0.9210905432701111, -0.8497033715248108, -0.1738535463809967, 0.28677698969841003, 1.5619351863861084, 1.3840616941452026, -1.1770703792572021, 0.7523881196975708, 0.9845254421234131, -1.4717851877212524, -0.2341994047164917, -0.7708227634429932, -0.6784034371376038, 1.3590182065963745, 0.9247375130653381, 0.2534027099609375, 0.49328798055648804, 1.111838459968567, 0.554652988910675, -0.11704710125923157, 0.7269892692565918, 0.1296604871749878, -1.9391024112701416, -1.791318416595459, -0.5448946356773376], [1.1185519695281982, -1.9319852590560913, 1.124847650527954, 1.137418508529663, -1.1919039487838745, -0.5760487914085388, -1.531194806098938, -1.0593831539154053, 0.5346230268478394, -1.3647499084472656, 1.0314313173294067, -1.9422684907913208, -0.48210105299949646, -0.21567171812057495, 1.4422041177749634, -2.4790735244750977, -0.32564762234687805, 1.2758153676986694, -1.5270596742630005, 0.0942096933722496, 1.0864243507385254, 0.6250988245010376, 0.18995825946331024, -1.8929383754730225, -0.3344515562057495, 1.4106082916259766, 0.04310945048928261, -0.05301722511649132, -1.5861669778823853, 1.0903074741363525, 1.2498953342437744, -1.2440260648727417, -0.31187403202056885, 0.8819552659988403, -1.3118441104888916, 2.1698944568634033, 1.203151822090149, 0.28255608677864075, -0.18355533480644226, 0.8728989362716675, 0.7910484075546265, -0.993095874786377, -0.6419212222099304, 1.6677770614624023, 1.0844907760620117, -0.15852701663970947, -0.04717385768890381, -1.0608538389205933, -1.0032225847244263, 0.016424031928181648, 0.14609205722808838, 0.7871793508529663, -0.6881442666053772, -0.17617955803871155, 0.03073657490313053, 2.6791319847106934, 0.9756802916526794, 1.587227463722229, 0.9536895751953125, 1.382349967956543, 0.32515454292297363, -0.9252306222915649, 0.0066605512984097, 0.7591037750244141], [0.18295696377754211, 0.7450900673866272, -0.9856223464012146, -1.819867491722107, 1.872565746307373, -0.7989007234573364, -1.0253973007202148, -1.1873425245285034, -0.9759429097175598, 0.7071321606636047, -0.11190297454595566, 1.7279831171035767, 0.024427639320492744, 0.2827231287956238, -0.15394818782806396, -0.5770184397697449, 0.021803567185997963, -0.5771470665931702, 0.5045288801193237, 0.6381508708000183, 0.7323423027992249, -0.5562379956245422, 0.13423439860343933, 0.25678908824920654, -0.2689495384693146, 0.056915827095508575, 3.114394426345825, 0.45500555634498596, -2.1154112815856934, -0.9258332252502441, -0.4011237919330597, 0.46732014417648315, 1.0121352672576904, 2.817443370819092, -0.4768524467945099, -2.1629843711853027, 2.7767391204833984, -0.11983482539653778, -0.6077213883399963, -0.09401895850896835, -0.8255782723426819, -0.6018217206001282, -1.7606478929519653, -0.18313021957874298, -0.460338294506073, -1.3192099332809448, -0.677008330821991, -0.556549608707428, 1.0132877826690674, -0.49488168954849243, 0.3024117052555084, -0.4874281585216522, -0.2591143250465393, 0.4246189296245575, 0.4092785120010376, 1.7135448455810547, 0.567301332950592, -0.957017183303833, -0.8037096858024597, -0.5159848928451538, 0.2394755482673645, 2.5528461933135986, 0.5767333507537842, 0.25308433175086975], [0.9499448537826538, -0.38944321870803833, -0.38737431168556213, -0.4539435803890228, 1.3150571584701538, -1.2352721691131592, -0.499222069978714, -0.8487836718559265, -1.2083855867385864, 1.037087321281433, 0.21503864228725433, -1.429168939590454, -1.865293264389038, -0.6135050654411316, 0.47813060879707336, 0.5112866759300232, -0.398767352104187, -1.423477292060852, -0.7001986503601074, 1.4721299409866333, 0.1759403645992279, -0.8290050029754639, -0.6968794465065002, 0.7386451959609985, 1.111172080039978, 0.04728826507925987, 0.5842700600624084, -1.2970364093780518, -1.473354458808899, -0.7700794339179993, 0.7554214596748352, -0.571814239025116, 1.5580922365188599, 1.4717352390289307, -0.5973922610282898, -1.246775507926941, -0.651023268699646, -0.8756488561630249, 1.1811782121658325, -1.3315410614013672, 0.4014815092086792, 0.1948843002319336, -0.6251464486122131, -0.25382399559020996, -0.2990787923336029, 1.7509145736694336, -0.772683322429657, -0.9124619364738464, 0.954954981803894, -1.1935921907424927, 1.1711935997009277, 0.06530267745256424, 0.11329040676355362, 0.1720592975616455, 0.46235892176628113, 0.35092005133628845, 1.129044771194458, 0.7983563542366028, 0.5400264859199524, -0.10568375885486603, -0.07089578360319138, 0.15831264853477478, 0.27340900897979736, 0.5615067481994629], [0.68623948097229, 0.23063653707504272, -0.6054311394691467, 0.9908164739608765, 0.025277305394411087, -1.7182567119598389, -0.48272523283958435, -0.3144119381904602, -0.6844092011451721, -1.4705479145050049, 1.8465701341629028, -0.572708249092102, 0.7119764089584351, -0.5786923170089722, 0.0004421246994752437, 1.4043357372283936, -1.3029365539550781, -1.4707387685775757, 0.3181179165840149, 0.09476743638515472, 1.4167391061782837, 2.2537949085235596, -0.21626277267932892, 1.103023886680603, -0.33326709270477295, -1.5703136920928955, -0.2716599106788635, 0.5985574126243591, -0.249332457780838, 0.1350049525499344, -0.2893527150154114, 1.1876407861709595, -0.876667857170105, -1.6096161603927612, -0.13667359948158264, 0.3795522153377533, 0.4451398551464081, 0.5792207717895508, 1.065578818321228, -0.5486491322517395, -1.8312822580337524, -0.4504612386226654, -1.678844928741455, 0.8629707098007202, 0.6280084252357483, -0.06779608130455017, -0.5397042036056519, 0.5706520080566406, -1.1136549711227417, -0.9431365728378296, 0.880585789680481, -2.2662770748138428, 0.04480421170592308, 0.2001502811908722, -0.29693883657455444, 0.14707356691360474, -1.203168511390686, -0.7994933724403381, -0.9294738173484802, 1.1929991245269775, -1.634450912475586, -1.1116923093795776, -2.1414968967437744, 0.2832871377468109], [0.5874466896057129, -1.4844398498535156, 0.8733869791030884, 0.34988048672676086, 0.43729105591773987, -0.14956499636173248, -0.08746938407421112, 0.1756507009267807, -0.24847370386123657, -1.5408034324645996, -0.04914439842104912, 0.06182963773608208, -1.3108066320419312, -1.017317771911621, -0.790141761302948, 0.2017022669315338, -0.2193879634141922, -0.07759758830070496, 1.8851152658462524, 0.4362128973007202, -0.700175404548645, -1.375952959060669, -0.22550331056118011, -0.6960285305976868, -1.343101143836975, -0.748634397983551, -0.2166629582643509, 0.5774314403533936, -1.556475043296814, 0.18622778356075287, -2.17366623878479, -0.514726996421814, 0.05577459558844566, -1.7748157978057861, -0.7543752789497375, -0.9888433814048767, -3.0134127140045166, 0.8757068514823914, -1.600245475769043, 1.2165192365646362, -0.6959517598152161, 0.7178345918655396, 0.48397696018218994, 0.21992933750152588, 0.6615034937858582, -0.1402328461408615, 0.36784952878952026, -1.3786007165908813, 0.37156182527542114, -1.9446319341659546, -0.733143150806427, 0.24452809989452362, 0.16828717291355133, 1.3337339162826538, 0.24200241267681122, -1.4378076791763306, 0.2240625023841858, 0.3216967284679413, 0.3091374337673187, -0.009256763383746147, -0.809232771396637, -0.5845270156860352, -0.701748251914978, -0.31665685772895813], [1.1084152460098267, -0.15230734646320343, -0.2328512817621231, 1.3507897853851318, 2.012017011642456, -1.9744176864624023, -0.35889339447021484, 1.1140669584274292, 2.426833391189575, 0.007750237360596657, 1.921000361442566, 0.11214527487754822, -0.20985329151153564, 1.4997814893722534, 1.5241773128509521, 0.35575631260871887, 0.9111913442611694, 0.7920511960983276, 0.6855929493904114, 1.8583277463912964, -3.4658596515655518, 0.2844970226287842, -1.4165263175964355, 0.7593355178833008, 0.09971262514591217, 1.6010308265686035, 1.225541591644287, -0.1427694708108902, -0.47131234407424927, -2.602198362350464, 0.4535488486289978, -0.37650546431541443, 0.17750944197177887, -1.4469752311706543, 0.9698340892791748, 2.2902262210845947, -1.4530446529388428, -0.003663428593426943, -1.0107120275497437, 1.0155216455459595, 0.1542893499135971, 0.5957298278808594, -0.30418282747268677, 0.05781709402799606, -0.7177944183349609, -0.38299041986465454, 0.7231429815292358, -1.589173674583435, 0.33164727687835693, -0.6544675230979919, 0.8362352848052979, -1.8538721799850464, 0.2437247633934021, 0.26038551330566406, 0.9072268009185791, -0.55713951587677, 0.4766167998313904, 0.2171919345855713, -1.5578675270080566, 0.17281237244606018, 1.602349877357483, 0.7551742792129517, 0.6388759016990662, 0.21260970830917358], [-0.7316038012504578, 1.0138113498687744, -1.1988822221755981, 0.1463496834039688, 0.682883083820343, 0.42006170749664307, 0.18903377652168274, 1.2894021272659302, -1.3594969511032104, 2.0038390159606934, -0.6844580173492432, -1.3837621212005615, 0.5260735750198364, 0.14527828991413116, 0.28729936480522156, -1.2510311603546143, -0.11905386298894882, 0.005777520593255758, -1.3764756917953491, 0.46407943964004517, 0.5475196242332458, 1.5514010190963745, 2.39654278755188, 1.4266589879989624, 0.4284321665763855, 1.153667688369751, -1.3827784061431885, 0.39523574709892273, 0.213838592171669, -0.0058792270720005035, -0.11053993552923203, -1.2184324264526367, 0.7694156765937805, 1.9531383514404297, -0.8382350206375122, 1.0924879312515259, -0.35201436281204224, 0.5275671482086182, 0.4921574592590332, -1.150252103805542, 0.30894213914871216, -0.056444160640239716, -0.6181900501251221, 3.070549964904785, -0.9768190383911133, -0.4350630044937134, 1.058936595916748, 0.21954044699668884, -1.4492347240447998, 0.8436803221702576, -0.2680146098136902, 1.0075433254241943, -0.06426649540662766, 1.031566858291626, -0.22741281986236572, -0.8627053499221802, 1.2136728763580322, 1.183457851409912, 0.8429057002067566, -0.22737161815166473, -1.726976752281189, 1.0245075225830078, -0.44264301657676697, -0.5406994819641113], [1.6016720533370972, 0.08305661380290985, 1.0604238510131836, -1.5691826343536377, -1.2984752655029297, -1.3836722373962402, -0.830413818359375, 0.8698045611381531, -0.4481465816497803, 0.7575790286064148, 0.42564842104911804, 1.3093675374984741, 2.073843002319336, 1.2041659355163574, -0.226309135556221, -2.2819344997406006, 0.0046069445088505745, -2.176471710205078, 1.0633398294448853, 0.7531306743621826, 0.4311387538909912, -0.5816674828529358, 0.16263048350811005, 3.2363624572753906, -0.3311360478401184, -0.9199806451797485, 0.7446100115776062, 0.005844769533723593, -0.4404292702674866, 0.24825307726860046, -0.20584963262081146, -1.0030940771102905, -1.5539653301239014, 0.5645805597305298, -0.7129160761833191, 0.978406548500061, -0.21473515033721924, 1.090547800064087, -0.5795546770095825, 1.9833087921142578, 1.1519538164138794, -1.561847448348999, -1.1867623329162598, -0.35837322473526, -0.48615315556526184, 1.276497483253479, 0.7030178904533386, 0.05418466404080391, 2.6678736209869385, 1.3486429452896118, 0.12867607176303864, 0.823657214641571, 0.7118631601333618, -1.619673252105713, 0.6261917948722839, -2.5389323234558105, -0.1905878186225891, -0.2597014009952545, 0.12012685090303421, -1.1924844980239868, 0.016926903277635574, 0.35617971420288086, -1.175194501876831, -0.04278433322906494], [-0.3953099846839905, 0.7092004418373108, -1.5434808731079102, -0.8674419522285461, -1.1554867029190063, -0.2974175810813904, -0.21716885268688202, 1.2029460668563843, 0.5914570093154907, -0.4573833644390106, 0.9098019599914551, -1.3482283353805542, -0.2812252640724182, -0.4571450650691986, 0.3987690508365631, -0.10067258775234222, -1.3363797664642334, -0.8850427865982056, -0.27303093671798706, 0.6496097445487976, -0.2175971418619156, -0.6595807075500488, -1.0419644117355347, 0.03526594489812851, 1.0171597003936768, -0.552265465259552, -0.6164528727531433, 1.0594463348388672, -1.3481208086013794, -0.7008395791053772, -0.526331901550293, 0.21116116642951965, 0.562630832195282, 1.4509592056274414, 0.06777779757976532, 0.6959744095802307, -1.2175458669662476, -0.7518449425697327, -0.3124667704105377, -0.09366090595722198, -1.3157627582550049, 0.16993920505046844, 0.638042688369751, 0.40673404932022095, -1.6776143312454224, -1.1445139646530151, 1.2915197610855103, 0.36248454451560974, 0.2534167170524597, 1.9653069972991943, -0.9858707189559937, 0.6457595229148865, 0.028695691376924515, 0.23242926597595215, -0.14978720247745514, 0.4530864357948303, -1.6404937505722046, 2.137502670288086, 0.9843738675117493, -0.16057631373405457, -0.9012648463249207, -0.7163920998573303, -1.6447595357894897, 0.47427016496658325], [0.3906145989894867, 0.5901502370834351, -1.5385775566101074, 0.6657002568244934, 1.5851166248321533, 0.9432916045188904, 1.2306787967681885, 0.5064293742179871, -0.9439763426780701, 0.1847868710756302, -0.6596915125846863, -0.06278875470161438, 0.11570578813552856, -0.6869502067565918, -0.271135151386261, -0.8427558541297913, 0.49156710505485535, -0.12711690366268158, -0.6272667646408081, 1.105920433998108, 0.9936665892601013, -1.5498768091201782, -0.9638279676437378, 0.7405207753181458, -1.5978436470031738, -0.12733913958072662, -0.14271992444992065, -1.0717915296554565, -1.0046800374984741, 1.1968345642089844, 1.6082608699798584, -0.11555701494216919, -0.08584415167570114, 0.47908687591552734, 0.07955107092857361, 1.2013821601867676, 1.487949013710022, -0.15616144239902496, -2.8900275230407715, -0.23031966388225555, 1.785664677619934, -0.7342711091041565, 0.7132261395454407, -0.7670959234237671, -0.7891668081283569, -1.2299535274505615, -0.28807875514030457, 1.4191581010818481, -1.260239839553833, -0.7153542637825012, -0.10964357852935791, -0.3041944205760956, 1.0381226539611816, -0.9232310056686401, -0.5837246775627136, -0.8398953080177307, 0.14621521532535553, -1.1457973718643188, -0.33244964480400085, -1.6373305320739746, -1.2456778287887573, -1.3751764297485352, -1.2956607341766357, 0.9863402843475342], [1.351277470588684, -1.1934943199157715, -0.7430861592292786, -0.5348511934280396, 0.9451718330383301, 0.4441717267036438, -0.6615598797798157, 0.19803838431835175, -0.9244236350059509, 2.1334526538848877, -0.08532898128032684, -1.4898325204849243, -0.36853086948394775, 0.7119127511978149, -0.6709569692611694, 0.33953192830085754, -0.6592519283294678, -0.34648874402046204, 0.9229766130447388, -0.11025732755661011, -1.4162211418151855, -1.970118761062622, -0.36739519238471985, 0.09260963648557663, 0.053261104971170425, 0.2538051903247833, -1.0533018112182617, -0.8746160864830017, 0.0800672322511673, -0.21476678550243378, 2.020373582839966, -0.08904990553855896, -0.7527101635932922, 1.0955957174301147, 0.86487877368927, -0.8038651347160339, 1.7860634326934814, 1.6520402431488037, -1.3775320053100586, 0.8167296051979065, -2.5382025241851807, -0.41529765725135803, -0.15503478050231934, 1.06840181350708, -1.8266626596450806, -0.2578010857105255, 0.2518431842327118, -0.029689781367778778, 0.5893626809120178, -0.41050079464912415, 0.683136522769928, -0.030692564323544502, 0.7993873357772827, 0.8754439949989319, 0.09917490184307098, -0.8408089280128479, 1.3498523235321045, -0.43120455741882324, 0.72823566198349, -0.11602102965116501, 2.701817035675049, -1.3248132467269897, -3.4345619678497314, -0.9198183417320251], [0.2632461190223694, -0.10849221795797348, 0.7347773313522339, 1.1452956199645996, -0.549115777015686, -0.5871251225471497, -0.3225587010383606, -0.17793215811252594, 1.3041516542434692, -0.37306228280067444, -0.9912652373313904, -0.09410843998193741, 1.5268139839172363, -1.501911997795105, 0.2910901606082916, 1.3554606437683105, -0.1837436705827713, -1.195876121520996, 0.4559487998485565, 0.023111188784241676, -0.819320797920227, -0.6945042610168457, -0.6336110234260559, -0.3091944456100464, -1.175679326057434, 0.7344431281089783, -0.5170900225639343, 0.5790228247642517, -1.431488037109375, 0.47650066018104553, -0.2384861409664154, -0.5923923254013062, 0.26873666048049927, 1.9243295192718506, 0.06777787208557129, 0.038021113723516464, 0.44716283679008484, -0.6741469502449036, -0.4964820444583893, 1.5343122482299805, -0.33030566573143005, 0.7083283066749573, -1.8890268802642822, -0.4306296706199646, -0.4815766215324402, -1.4231815338134766, 0.5960448384284973, -1.112473964691162, -1.2127012014389038, 0.9554783701896667, 0.7959704995155334, 1.017466425895691, 0.44604796171188354, -0.8140949010848999, -2.651190996170044, 0.7353675365447998, -0.5248979926109314, -0.6447088122367859, -0.336433082818985, 0.05684651434421539, 1.8940550088882446, 0.687289834022522, -2.2785637378692627, -0.05292590335011482], [0.6811161041259766, -1.2595906257629395, 1.9167280197143555, 0.21051962673664093, 0.0737980380654335, 0.1536092460155487, 0.859544038772583, 1.3081778287887573, 0.6764739155769348, -0.4479379653930664, -1.441545009613037, -0.9671750068664551, -0.3747449219226837, -0.9249204993247986, -1.0378855466842651, -0.9014923572540283, 2.0067787170410156, 1.5022356510162354, 1.025205135345459, -0.5858094692230225, 0.14988236129283905, -0.2784285247325897, 1.9870316982269287, 0.053523045033216476, 0.2916591763496399, 0.05402660742402077, -0.7913346290588379, 0.17055045068264008, 0.7160615921020508, 0.20950010418891907, 0.9495258927345276, 1.6358294486999512, 2.052433729171753, -0.2454020082950592, -1.0397337675094604, -0.5497534275054932, 0.2925786077976227, 1.079450011253357, -0.726645290851593, -0.0517628975212574, -0.9245005249977112, 1.026989459991455, -0.7867440581321716, -1.0495197772979736, -0.7944715619087219, 0.4667966663837433, -1.8010090589523315, 0.8241254091262817, 1.330726981163025, -0.9079257249832153, -0.981132447719574, -0.6662248373031616, 0.7425466775894165, 1.354414939880371, -0.4862714111804962, -0.14644783735275269, 0.5163270831108093, 0.6051504015922546, 0.12186219543218613, -0.16542373597621918, -0.9843102097511292, 0.015517542138695717, 1.205556035041809, -0.6330706477165222], [0.6614906191825867, -0.18478524684906006, 0.07059483230113983, -0.01674218662083149, 0.22381684184074402, -0.9871382117271423, 0.7236858606338501, 0.6297917366027832, -1.5037575960159302, 0.25777167081832886, -0.2712997794151306, 0.24994485080242157, -2.8043789863586426, -1.1954026222229004, -0.19039088487625122, -0.301802396774292, 0.25983521342277527, -1.7154055833816528, 0.15960150957107544, -2.0965802669525146, 0.8462632298469543, -0.032969880849123, -0.8531903624534607, 0.6736464500427246, 1.6441065073013306, -1.5075116157531738, -0.2151719331741333, 0.5556307435035706, 0.3534899652004242, -0.005770609248429537, -0.4588812589645386, -0.044616371393203735, 0.5153105854988098, -0.2800654172897339, -0.16846205294132233, 2.3451738357543945, 1.3290647268295288, 0.05040210485458374, -0.25990810990333557, -0.05877092853188515, -0.7611942887306213, 0.9955097436904907, 0.10820799320936203, 0.1096765547990799, -0.09929080307483673, 1.2739057540893555, 0.1878492385149002, 1.2865419387817383, -0.03289918228983879, -0.9729244709014893, -0.46359947323799133, -2.6353673934936523, 0.9285569190979004, -0.056000005453825, -1.014838695526123, 0.13727979362010956, -0.6226900815963745, -1.0169867277145386, 0.2203824520111084, -0.4416358172893524, -0.06534161418676376, -0.08559180051088333, -0.6998850703239441, -1.028022050857544], [-0.004701745230704546, -0.23586933314800262, 0.7612292766571045, -0.37386393547058105, 0.9706245064735413, 0.5340889096260071, 1.391918659210205, 1.2124855518341064, 0.36739006638526917, -0.5912903547286987, 0.5134183764457703, 1.2230913639068604, 0.36547571420669556, 1.04117751121521, 0.5368817448616028, 1.4912512302398682, -0.2749450206756592, -0.9416079521179199, -1.2552043199539185, -0.05702536553144455, -1.0781419277191162, -2.4496140480041504, 0.08430812507867813, 1.3891886472702026, -0.91730797290802, -0.32399171590805054, -0.5638182759284973, -0.0464508943259716, 0.964604377746582, -0.4701329469680786, -1.3276691436767578, 0.16290757060050964, 1.269831895828247, -0.3933834731578827, -0.4246601462364197, -0.5058608651161194, 0.22273987531661987, -1.285606861114502, -0.19613902270793915, 0.8339989185333252, -0.12277563661336899, 1.0210720300674438, 0.2033654749393463, 1.0454283952713013, 0.6337490081787109, 0.01865277998149395, 0.08374466001987457, 1.7610702514648438, 1.7072547674179077, 0.1840803623199463, 1.1847866773605347, 1.4176158905029297, -0.5496537089347839, 0.06794557720422745, 2.273909568786621, -0.7544667720794678, -0.037654463201761246, -0.9143700003623962, -0.10627809911966324, -0.4366402328014374, -0.7949119210243225, 0.7668439149856567, -0.9636023640632629, 0.11353753507137299], [-0.5140092372894287, 0.8424922227859497, -0.03819998726248741, -2.407363176345825, 0.629949688911438, -0.05191629007458687, 0.02983464114367962, 1.0772980451583862, 2.0653960704803467, -2.5177712440490723, -3.259403944015503, -0.3257400393486023, 1.1514813899993896, 0.05180605873465538, 0.3470267355442047, 0.2875693738460541, -1.0053867101669312, -1.1167089939117432, -0.4358447790145874, -0.11626289784908295, 0.9147059321403503, 0.14502449333667755, -0.4902821481227875, 1.270828127861023, -0.6087017059326172, -0.4204340875148773, -0.86188143491745, -0.06802840530872345, 0.9300224781036377, -0.38285401463508606, -1.3555828332901, 0.8752809166908264, 2.830923557281494, 0.7259393930435181, -1.899658203125, 1.5619761943817139, -0.4974580407142639, -0.01890336349606514, 0.2704736292362213, -0.6147235035896301, 0.3071889579296112, -1.445685863494873, -0.6895991563796997, 0.5324959754943848, 1.1804193258285522, 0.17599964141845703, -0.9495783448219299, 2.040778636932373, 0.7206748127937317, 1.1935173273086548, 0.4060315489768982, 1.2131227254867554, -0.0672842487692833, -1.1095466613769531, 0.13986602425575256, -1.4642820358276367, -1.1748610734939575, 1.3522921800613403, 0.28912290930747986, 0.7439286708831787, 1.0313910245895386, -1.330759048461914, -0.24242332577705383, -1.0942949056625366], [-0.11431250721216202, -1.0370815992355347, -0.4600363075733185, -0.6332942843437195, -0.6187142133712769, -1.4984315633773804, 1.4294108152389526, -0.025395233184099197, -0.033982131630182266, -0.07286487519741058, 1.0447138547897339, 0.1403375118970871, -0.3674914836883545, -1.173351526260376, -1.0433189868927002, -1.0004693269729614, 0.12002062797546387, 1.604298710823059, 0.022118929773569107, 2.9947714805603027, 1.0696966648101807, -0.6249744892120361, 0.9981518983840942, 0.616219162940979, -1.687666654586792, 1.0527782440185547, 1.44661283493042, 1.571773648262024, 0.44702252745628357, -0.1462002694606781, 0.7063578963279724, 0.48788031935691833, 0.7809619307518005, -0.4896879494190216, 0.5319965481758118, -1.0510307550430298, -0.0782051607966423, 0.8910694718360901, 1.285854458808899, 0.612792432308197, -0.747642457485199, 0.0894780158996582, 1.2189055681228638, 0.3594099283218384, 0.10264412313699722, 2.330479145050049, 0.47069498896598816, -0.41437897086143494, 2.4752161502838135, 0.29891282320022583, 0.1918860673904419, -0.321927547454834, -0.6988756656646729, -0.9904963970184326, -0.578639030456543, 0.6059674024581909, 1.1112819910049438, 1.7682231664657593, 0.19160446524620056, 0.7392317056655884, 0.05595042556524277, -0.4683573544025421, -0.5140317678451538, -1.2458348274230957], [0.33869463205337524, 0.1497378796339035, -0.4247216582298279, 0.6989601850509644, 1.0695793628692627, 0.08159526437520981, 0.02698519453406334, -0.8821554183959961, 1.3105219602584839, -1.640967607498169, -0.4248913526535034, -0.4062826633453369, 0.15785472095012665, 0.8345494866371155, -0.16932034492492676, -1.8797290325164795, 0.9408966302871704, 0.6721340417861938, 1.3387260437011719, -0.9847633242607117, -0.7315347194671631, -0.637924075126648, 0.24327091872692108, -0.9257306456565857, 0.2748456299304962, -1.066447377204895, 0.5196324586868286, 0.9533050060272217, -0.2507394850254059, -0.7287744879722595, 0.6086363196372986, -1.234022855758667, 0.34275928139686584, -0.10531643033027649, 0.33879756927490234, 2.221932888031006, 1.1045989990234375, -0.9075157046318054, -0.7934516668319702, 0.37510016560554504, 0.2386505901813507, -0.415653795003891, -1.5133641958236694, 0.024723555892705917, 1.8245766162872314, -1.2723215818405151, 0.3783445358276367, 0.46238481998443604, -0.2040136307477951, 0.07583515346050262, -0.6161900162696838, 0.06919374316930771, -1.5680829286575317, -0.7049412131309509, 1.2477085590362549, -0.23622290790081024, 3.2598001956939697, 1.0985257625579834, 0.3591035008430481, 1.102542519569397, 0.7066205739974976, -1.0546627044677734, 0.0783969834446907, -0.7626621723175049], [-0.9387498497962952, 0.9119107723236084, -0.15673886239528656, 0.3428996205329895, 0.15017439424991608, 2.0791001319885254, -0.19461333751678467, 0.44939398765563965, -1.2803634405136108, 0.017908206209540367, -0.6146444082260132, 1.0046162605285645, 1.5837159156799316, -1.9365493059158325, -0.7773675918579102, -0.5511468052864075, 0.44676071405410767, -0.36947470903396606, 0.22014908492565155, -2.0095624923706055, -0.12085933238267899, 0.6374235153198242, -0.6113720536231995, -0.3654947578907013, -0.944855809211731, -0.7403873801231384, -0.6993039846420288, 1.5186421871185303, -0.5607534646987915, -0.6507804989814758, -1.246029257774353, 0.46427643299102783, 2.4563803672790527, 0.1005735993385315, 1.896284580230713, 1.4113343954086304, -0.6583667993545532, -1.0002721548080444, 0.5955234169960022, 0.8147056102752686, 0.30237504839897156, 0.39740052819252014, 1.8359878063201904, 0.9415048956871033, 0.9680522084236145, -0.0505174845457077, 1.566770076751709, -0.6145118474960327, 0.14225704967975616, -0.8798763155937195, 0.35175517201423645, 0.816961944103241, 1.6370692253112793, -0.09228697419166565, 0.8424569964408875, 0.6039350032806396, -0.8561106324195862, 0.5795746445655823, -0.05772847682237625, -0.6942124962806702, 0.14539727568626404, 0.7174273133277893, -0.757408618927002, 2.5478708744049072], [0.039331287145614624, 1.7509845495224, -1.9691686630249023, -0.5998086929321289, 1.398235559463501, -2.8102805614471436, -0.18496818840503693, 1.1777368783950806, 0.6592345833778381, 0.35244157910346985, 1.2986987829208374, -0.5341089367866516, -1.062864065170288, -1.4440312385559082, 0.4771653115749359, 0.7099095582962036, 1.0475444793701172, 0.26748940348625183, 0.5198829174041748, 0.0799100399017334, 0.025290509685873985, -0.3659258484840393, -1.4612637758255005, 0.7440811991691589, -0.8196194767951965, -0.3269555866718292, -0.029297279193997383, 1.8666340112686157, -0.1122281402349472, -0.03083035908639431, 0.4075792729854584, -0.8132336139678955, -0.06820899248123169, -0.17087221145629883, -0.1903814971446991, 0.11703222244977951, -0.034781865775585175, -1.7136476039886475, 0.5712399482727051, 1.0803520679473877, -1.3166922330856323, -0.9938111305236816, -0.7662678956985474, 0.12263596057891846, -1.6192060708999634, 0.285491019487381, 0.6691519618034363, -1.3423391580581665, -1.416735053062439, 1.5079072713851929, 1.4502887725830078, 0.04321477189660072, -0.11786860972642899, 0.6233901977539062, -1.0235439538955688, -0.09508109837770462, -0.320739209651947, -1.3789392709732056, -1.34731125831604, 1.610207200050354, -1.5562342405319214, 0.9313243627548218, -0.5140222311019897, -0.977377712726593], [-1.3872432708740234, -0.21850113570690155, -1.3191585540771484, 0.8645486831665039, 0.8261331915855408, 2.1172499656677246, 0.766615629196167, 1.4796382188796997, -0.23487623035907745, 0.9153710603713989, 0.5722206830978394, 0.8404674530029297, -1.0856462717056274, -1.2636020183563232, -0.5259630084037781, 1.0137141942977905, -0.7024341821670532, 0.6900988817214966, -0.11854634433984756, -0.03261634707450867, 0.24733325839042664, 0.5667696595191956, 1.5174862146377563, -1.1551239490509033, 0.8047434687614441, -1.4629367589950562, 1.8136714696884155, -0.3740849196910858, -0.5802305340766907, 0.7971214056015015, 0.4787384867668152, -1.4217803478240967, -1.3093693256378174, -0.2248854786157608, -0.8973453640937805, -0.2821279466152191, -0.8478220701217651, 1.0492241382598877, -0.10243713110685349, -1.2419297695159912, -0.3607093095779419, -0.07990846037864685, 0.27630800008773804, -0.05847936123609543, -1.4144885540008545, 1.1542184352874756, -0.7907553911209106, -1.8517836332321167, 0.5026071071624756, 0.7178568243980408, 2.7387900352478027, -1.2725733518600464, 1.5390971899032593, 1.0742093324661255, -1.3314725160598755, -0.5474935173988342, -1.3059194087982178, -0.6817043423652649, 1.3273868560791016, 1.0537476539611816, -0.1710522174835205, 0.313885897397995, -0.5655722618103027, -0.5179557204246521], [0.5429276823997498, -0.2717629671096802, -0.5899307131767273, -0.5137585997581482, 1.2760543823242188, 1.6623555421829224, 0.27588868141174316, -0.7956767678260803, 0.3982333838939667, 1.5936110019683838, -0.7779149413108826, 0.13010580837726593, -0.12524715065956116, 1.3112695217132568, -1.646637201309204, -0.8016712665557861, 0.32826271653175354, 1.9421387910842896, -0.21614418923854828, -0.6158207654953003, -1.795940637588501, 1.1739894151687622, 1.7430307865142822, 1.211796522140503, -0.2541882395744324, 0.059724416583776474, -0.4526042938232422, -0.7957875728607178, -0.9675156474113464, 0.40251192450523376, 1.5354598760604858, -0.3810670077800751, -0.22781424224376678, -0.09644743800163269, -1.9348723888397217, -1.3159977197647095, -0.2759218215942383, 0.6885201334953308, 0.5407027006149292, -0.3688223958015442, -0.4414321184158325, -2.5468218326568604, -0.2720484435558319, 0.47948065400123596, -1.4447168111801147, 1.09921133518219, 0.8422040939331055, 0.2509306073188782, 1.1863895654678345, 0.939983606338501, 0.048447541892528534, 0.06530014425516129, 0.00523435277864337, -0.10921967029571533, 0.32786187529563904, -0.8013489246368408, -0.5695464611053467, 0.017871133983135223, 1.200914740562439, 0.7156421542167664, 0.8876184821128845, 1.2066775560379028, -0.9680519700050354, -1.1716396808624268], [-0.5387363433837891, -1.7516160011291504, -0.33076828718185425, -0.34623757004737854, -1.6858646869659424, -0.8705757856369019, -0.2740742564201355, 0.8521080017089844, 1.8727213144302368, 0.9752894639968872, 1.2580199241638184, -0.14122802019119263, 0.3072526752948761, 0.9210314154624939, -0.36907631158828735, 0.27993133664131165, 0.7334949970245361, -0.7161184549331665, -0.6711480021476746, -0.7004839777946472, -0.017203954979777336, -0.5243653059005737, -1.5857211351394653, -0.5426502823829651, 0.44527745246887207, 0.05233190208673477, 0.04063466563820839, -0.46048593521118164, 0.3126097023487091, 1.3517578840255737, 0.11004568636417389, 0.10845759510993958, -1.1766810417175293, 1.065346121788025, -0.19422248005867004, -1.2571208477020264, -1.1114789247512817, 0.833026111125946, 0.4902164340019226, 1.4881724119186401, -0.05002204328775406, -0.7673091292381287, 0.2700003981590271, 1.1326547861099243, -0.8777831792831421, 1.1314245462417603, 0.803135097026825, -1.5826467275619507, 0.5583010315895081, 2.0756897926330566, -0.645561695098877, -0.38674619793891907, -2.157015323638916, 0.8349030613899231, 0.07296612858772278, 0.2695106863975525, 0.9256783723831177, 1.7543805837631226, 0.24675065279006958, -1.4496617317199707, 0.5147765278816223, 1.2488672733306885, 0.7799733877182007, 0.6797018051147461], [-0.9427043795585632, 1.6170332431793213, 0.5492225885391235, -0.06498029828071594, 0.3251199424266815, -0.7693171501159668, -0.6533524394035339, 0.9236258268356323, -0.7891947627067566, -0.5253111720085144, -0.21386350691318512, 0.5419999361038208, -0.3242814838886261, -0.523243248462677, -0.8360828757286072, 0.8393927216529846, 0.8778477907180786, -1.6952868700027466, 1.1398817300796509, 1.0921638011932373, -1.7852234840393066, 1.026228904724121, 0.13156342506408691, 0.03358403965830803, -0.9167569875717163, 0.9457329511642456, -1.1397837400436401, -0.055632222443819046, -0.17916959524154663, -1.1084942817687988, -1.554738998413086, -1.8417779207229614, -0.5491757392883301, 1.1689538955688477, -1.3075392246246338, 0.8229935765266418, 1.3148828744888306, 2.1028761863708496, 0.8970311880111694, -0.6180917024612427, 0.49589747190475464, 0.011511590331792831, 1.1287786960601807, -0.07229287922382355, -1.1960002183914185, -1.2560064792633057, -0.3025169372558594, -0.4998176693916321, 0.16244561970233917, 0.24135449528694153, -0.5169775485992432, -1.2920103073120117, 1.604801893234253, 0.7437864542007446, 1.4029641151428223, -1.3451937437057495, 1.6689826250076294, 0.6801773905754089, 2.282423734664917, 1.2774089574813843, -1.556931495666504, 1.4338544607162476, 0.6841668486595154, -0.9508569836616516], [0.27016007900238037, 0.7916960716247559, -0.7489816546440125, 1.3757635354995728, -1.1328554153442383, 0.559567391872406, -0.22616317868232727, -1.7586930990219116, 1.7691874504089355, -1.199538230895996, -1.2697768211364746, -0.5445004105567932, 0.7329151630401611, 0.13825368881225586, 0.9609773755073547, -2.3405187129974365, 0.7975361943244934, -0.7421668171882629, 0.01771206595003605, -0.5907399654388428, 0.1481759399175644, -0.9561766386032104, -0.6756817102432251, -0.9609737396240234, -1.279210090637207, -1.5783265829086304, 1.7779184579849243, 1.443717122077942, -1.5823512077331543, 0.7015488743782043, 0.8433681130409241, 1.1496145725250244, -1.6981276273727417, -1.0166351795196533, 0.13773097097873688, -0.976691484451294, -0.11710894107818604, -0.048197608441114426, 1.8641939163208008, 0.16577942669391632, -0.46485838294029236, 1.7203818559646606, 0.1854509711265564, -0.28557559847831726, -0.2323285937309265, 0.552551805973053, 0.07348654419183731, 0.46359115839004517, -0.15559236705303192, -1.30299973487854, 0.9912288784980774, 0.1693405658006668, -0.19838352501392365, -0.5498767495155334, -0.191012442111969, -0.0026713714469224215, -0.1279737800359726, 0.2399701625108719, -0.9983730912208557, -0.3267001509666443, 1.1839872598648071, 0.5506575107574463, -0.21369817852973938, 0.8912695050239563], [1.3895010948181152, 0.4456096589565277, -0.8479325175285339, -0.3972695767879486, -0.3514769375324249, 0.10115431994199753, -1.334614872932434, -0.343927264213562, 0.9445083737373352, 1.8364709615707397, 1.7126522064208984, -2.142714023590088, 1.04644775390625, -0.5454211235046387, -0.8776177167892456, 0.8864263892173767, -0.5665534138679504, -0.47929006814956665, 0.3821718990802765, 0.7784214615821838, 0.7378615140914917, 1.4397423267364502, 0.6578430533409119, -0.0829884260892868, -1.3637454509735107, 0.553551971912384, -1.4015971422195435, -0.6491614580154419, -0.6293595433235168, 0.4860885739326477, 0.8856406807899475, -0.1323135644197464, -0.23842215538024902, -0.17843353748321533, -0.2745533585548401, -3.418790817260742, -0.6478319764137268, -1.3885210752487183, -0.7680726051330566, 1.7130569219589233, -0.27201205492019653, -0.05301773175597191, 0.4454231858253479, -0.5209384560585022, 0.5739762187004089, -1.4027541875839233, -1.2272449731826782, 0.4196532368659973, 1.6788944005966187, 0.4907478094100952, -0.20321401953697205, -0.156587153673172, -0.545169472694397, 1.9368133544921875, 1.9306187629699707, 0.17533215880393982, -0.3503442108631134, -0.18129095435142517, -0.7223849296569824, 0.8588791489601135, 2.30373477935791, 0.8027155995368958, -0.5171083807945251, 1.8850773572921753], [-1.8224127292633057, 1.3361440896987915, 0.19978027045726776, 1.9505168199539185, -2.5197463035583496, -0.1349572241306305, 0.5097203254699707, 0.029979700222611427, -0.8707417249679565, -0.6856105923652649, -0.0027384848799556494, -0.9424973130226135, -0.9330251216888428, 1.3474596738815308, 0.9829221367835999, -0.5286763906478882, -0.06574670225381851, -0.3316721022129059, -1.670593023300171, 0.943930447101593, 0.28301453590393066, 0.1344507485628128, -0.9769859313964844, 0.146290585398674, 1.9513452053070068, -1.1502492427825928, -0.20770703256130219, -1.414229393005371, -1.7487210035324097, -0.6129113435745239, -0.08045965433120728, 0.24368423223495483, -1.6709933280944824, -1.2556558847427368, -1.4132091999053955, 0.2937565743923187, 1.6809797286987305, -2.514275074005127, 0.6145692467689514, -0.7964445948600769, 0.410433292388916, -0.9980892539024353, 0.8449320197105408, 0.5932894349098206, 0.540368914604187, -2.121535301208496, -0.46227651834487915, 0.3021985590457916, -0.03887457028031349, -0.382687509059906, -0.07621410489082336, -0.4583902359008789, -1.3007619380950928, 0.03829953819513321, 0.7629157304763794, 0.45604416728019714, 0.17081880569458008, 0.11495866626501083, 0.6970328688621521, -0.44662898778915405, -0.7089503407478333, -0.5088086128234863, 1.0104446411132812, 1.082027554512024], [0.5276479125022888, 2.024221897125244, 0.48509591817855835, 1.0445386171340942, -0.37025347352027893, -0.506557822227478, 1.090606451034546, 0.4838387370109558, -0.6533756852149963, 0.08931829780340195, -0.06626851856708527, 0.24700641632080078, -0.9061276316642761, -1.40866219997406, -1.5545369386672974, -0.5200386047363281, -0.9748611450195312, -0.40801337361335754, -0.13637441396713257, 0.18683837354183197, 1.4436125755310059, -0.9648411273956299, -1.0517659187316895, -0.23382338881492615, 0.9761633276939392, 0.35534143447875977, -0.28963503241539, -0.48025786876678467, -0.7783699631690979, -0.05201956629753113, -0.09890241175889969, -1.8243684768676758, -0.1741480827331543, -0.002452665474265814, 1.1859418153762817, 1.1442008018493652, 0.6228311657905579, -0.3008642792701721, 1.334638237953186, 0.49129217863082886, 0.6326284408569336, 0.6709383726119995, -1.0158164501190186, 0.7501661777496338, 0.3985430598258972, 0.39009547233581543, 0.1511000394821167, -0.39910703897476196, -0.24050533771514893, -0.5589686632156372, -0.9971315860748291, 0.45308345556259155, -2.047614336013794, 0.836600124835968, 0.32196953892707825, 0.9118509888648987, 0.34774067997932434, 0.26121285557746887, 0.04783357307314873, -0.968685507774353, 0.7715829014778137, -1.5318025350570679, -0.5730258822441101, -0.12449940294027328], [-0.7266321778297424, 1.934685230255127, -0.08757449686527252, 2.056448221206665, 0.4593198895454407, -1.6465363502502441, -0.14105601608753204, -0.14103247225284576, -0.47648781538009644, -0.5546255111694336, -0.7275713086128235, -0.9283018708229065, 0.6673415303230286, 1.7326430082321167, 0.8374320268630981, -0.31536829471588135, 2.499624729156494, -1.3162046670913696, 0.39347681403160095, -0.4702877104282379, -0.33736228942871094, -1.3176215887069702, 1.0221121311187744, 0.554460346698761, 0.497627854347229, -0.6537073254585266, 0.7388843297958374, 0.9680058360099792, -0.6245275139808655, -1.2527964115142822, -0.30253469944000244, 0.2643779516220093, 0.30806902050971985, 0.20985150337219238, 0.3940962255001068, -0.6588248014450073, -1.1311496496200562, 0.003706190502271056, 0.22005639970302582, 0.005433085840195417, -0.036113590002059937, 0.5549141764640808, 0.6980209946632385, -0.08458135277032852, 1.4846559762954712, -0.24387040734291077, -1.6052253246307373, 2.4450573921203613, 0.05790344998240471, 0.7685215473175049, -1.4846022129058838, 1.5487056970596313, 0.2708872854709625, 1.2473978996276855, -0.9302523136138916, -0.012126595713198185, 0.21264073252677917, 0.5031642317771912, 1.0318406820297241, 0.28223007917404175, 1.2506861686706543, 0.32775643467903137, 0.7617133259773254, 0.24654191732406616], [0.28145506978034973, 0.29821714758872986, -0.25663435459136963, 0.6374134421348572, -0.26474472880363464, 0.90805584192276, -0.7013077139854431, 0.5493731498718262, 0.6338766813278198, 1.3976553678512573, 0.06631883978843689, 1.1517491340637207, -1.3176065683364868, 1.8866324424743652, 1.4987608194351196, 0.36185958981513977, -0.6154645681381226, -1.3176963329315186, 0.2724079489707947, -0.046011172235012054, -0.3213464319705963, -0.30717340111732483, -1.1788411140441895, 1.2469767332077026, 1.3212504386901855, 0.15183189511299133, -1.4252171516418457, 0.5089002847671509, 0.3179575204849243, -1.3571689128875732, 1.6857670545578003, -1.5572766065597534, 0.34236741065979004, 0.1668289601802826, -0.7161679863929749, -2.2199363708496094, -0.1766638308763504, 0.6335138082504272, 0.07820170372724533, -1.8666619062423706, -0.42054426670074463, 0.23850613832473755, -0.2187652289867401, 1.112960934638977, -0.6382288932800293, -1.5029230117797852, -0.09590321034193039, 0.6048456430435181, -0.41442495584487915, -1.3891690969467163, 0.1346108466386795, -0.36589139699935913, -1.2809841632843018, -0.08179683238267899, -0.32040420174598694, -1.9240795373916626, -1.1763179302215576, -0.2107405811548233, 0.11291804909706116, 0.7033456563949585, 0.4068695306777954, -0.618507981300354, 1.4798588752746582, 0.2809261977672577], [-0.6363177299499512, 0.1311100572347641, -0.08845680952072144, -0.8888130187988281, -0.8467148542404175, 0.46762412786483765, -0.16895154118537903, -2.329340696334839, -1.0919764041900635, 0.7308741807937622, 1.8404346704483032, -0.5172985792160034, -0.5609828233718872, -2.1548030376434326, 1.5909708738327026, -0.9767093658447266, -0.7903710007667542, 1.578802227973938, 0.03261696174740791, 0.6657660007476807, -0.171372190117836, 0.45937442779541016, -0.727246105670929, 1.0061092376708984, -1.7892990112304688, -1.8030588626861572, 0.47017940878868103, 1.7772265672683716, 0.016031581908464432, -0.5176001787185669, 0.7773217558860779, 0.40652069449424744, 0.4239279329776764, -1.9573546648025513, -1.0881088972091675, -0.9296536445617676, 1.5359089374542236, 0.06851135194301605, -0.41537341475486755, 0.4568801820278168, 0.4465940296649933, -0.5404372215270996, -0.47918644547462463, 2.390733480453491, -1.7529939413070679, -0.4727494418621063, 1.7883106470108032, -1.6305538415908813, -0.11367880553007126, 0.12183445692062378, -0.2962675094604492, -0.6518325805664062, -0.23872478306293488, 1.337027668952942, 0.022405697032809258, 1.0760070085525513, 0.47279196977615356, -0.2757152020931244, -0.671382486820221, 1.4556652307510376, -0.8036208748817444, 0.7860713005065918, -0.3755320906639099, -0.1779600977897644], [-1.440587043762207, -1.0274912118911743, -1.858304738998413, -1.3625373840332031, -0.531943142414093, -0.07974296808242798, -0.890426516532898, -0.18198564648628235, 1.1222105026245117, -0.09739743918180466, 1.0438942909240723, -0.2725825011730194, 0.11048182845115662, 2.546138048171997, 0.3509986400604248, 0.49351224303245544, 1.2040036916732788, 1.2512571811676025, -2.5881519317626953, -0.9334794878959656, -0.8370562195777893, -0.019210871309041977, 0.6551055908203125, -0.4223882853984833, -0.11654478311538696, -0.8314783573150635, -0.074522465467453, -1.759792447090149, 0.10228178650140762, -0.35480478405952454, 0.7212228178977966, 1.4378913640975952, -0.9893349409103394, -1.4349966049194336, 0.8796639442443848, -0.717346727848053, -0.8711820840835571, 0.007470869459211826, -0.2535623610019684, -0.5425994396209717, 1.5458409786224365, 1.69937264919281, 2.215928316116333, -0.39215588569641113, 1.075439214706421, -0.5693215727806091, -2.4161388874053955, -1.7584662437438965, -0.06385792046785355, -1.35085129737854, -1.071196436882019, 0.7417860627174377, 1.0822330713272095, 1.1251546144485474, -0.701179027557373, 1.249833345413208, -1.0182024240493774, 0.6860318183898926, 0.2828616797924042, 0.5506558418273926, 0.06892802566289902, 0.31441766023635864, -1.8050987720489502, 1.7133150100708008], [1.018925666809082, 0.9281589388847351, -0.45792773365974426, -0.5673422813415527, -0.061793725937604904, -1.6156537532806396, 1.1125236749649048, -0.07913920283317566, -1.3873080015182495, -0.37668272852897644, -0.3413143754005432, 0.307947039604187, 0.13689236342906952, 0.9862985014915466, 0.359662264585495, -1.8454089164733887, -1.0283830165863037, 0.356220543384552, -1.094797134399414, 0.7753096222877502, -0.962889552116394, -0.5480560660362244, 0.5081251859664917, -0.4545169174671173, 0.3267350196838379, 1.305384635925293, -2.6783406734466553, -0.8587571382522583, 2.3400495052337646, -0.17614628374576569, 0.7358909249305725, 0.5062621235847473, -1.0948039293289185, 0.8761455416679382, 2.251441240310669, 1.1545668840408325, -1.1583305597305298, 0.1730552464723587, -1.567456841468811, 1.1676955223083496, -1.2628570795059204, -0.765780508518219, 0.5871848464012146, -0.7275498509407043, -0.18831035494804382, -1.1378597021102905, -1.444356083869934, -1.341495156288147, -0.5740116238594055, -0.39032915234565735, 1.1595609188079834, -1.5539402961730957, -0.2251170575618744, 1.1229057312011719, 0.46733734011650085, 1.3198859691619873, 1.2574248313903809, 0.8426432609558105, -0.7107294201850891, 1.9323391914367676, -0.5317474007606506, -1.0531035661697388, 1.1192833185195923, 1.350359320640564], [1.6161409616470337, 0.31476229429244995, -0.4264318645000458, 0.05473826080560684, -0.6089571118354797, -1.0701173543930054, -0.024203602224588394, 0.4864683449268341, -1.7059929370880127, -0.639221727848053, 1.5274689197540283, 0.1329650729894638, 1.3162020444869995, 0.013929398730397224, 0.049608610570430756, -0.7658683061599731, -0.25512245297431946, 2.0940663814544678, -2.035628080368042, 0.10852041840553284, 0.8725490570068359, -0.05766536667943001, 0.48864084482192993, -1.2951287031173706, 0.8893040418624878, 1.3343393802642822, 3.0074801445007324, -0.1975887268781662, 0.2930137515068054, 0.16013893485069275, 1.1810764074325562, -1.1492804288864136, 2.5084829330444336, 0.45749908685684204, 0.8740654587745667, -1.0332120656967163, 0.31525832414627075, -0.4464038610458374, -0.9607675075531006, 0.7776058912277222, 0.5520060062408447, 1.193455457687378, 0.046390656381845474, 0.0268122348934412, 0.7687821984291077, 1.2527422904968262, -0.28374186158180237, 1.8204925060272217, 0.4014124274253845, -0.2639935314655304, 1.4464994668960571, 2.3763959407806396, 0.4053226411342621, 1.4877729415893555, -0.34348320960998535, 0.9433726668357849, 0.5978605151176453, -0.58899986743927, -0.7824263572692871, -0.9366092681884766, -0.15180091559886932, -1.2850815057754517, -0.01738796755671501, 0.2183661311864853], [-0.37105125188827515, -1.377482533454895, -0.5866331458091736, -0.42358818650245667, -0.773777425289154, -0.20251062512397766, -1.1730798482894897, 0.21564631164073944, -2.334132194519043, -0.19128304719924927, -0.6089969277381897, 1.46721613407135, -0.0015111052198335528, 0.41278406977653503, -0.2334362268447876, -0.39163610339164734, 0.0782289132475853, -0.30019789934158325, -0.20944258570671082, -1.3047975301742554, 0.33552202582359314, 0.9747862815856934, 1.3170154094696045, 1.2321172952651978, -1.6224839687347412, -0.37582096457481384, 1.162713885307312, 0.8434430360794067, -0.31282827258110046, 0.2752196192741394, 0.4600226581096649, 0.8057990670204163, 0.08454547077417374, -1.3921726942062378, 0.8388456702232361, 1.3960291147232056, -0.2934584617614746, 0.5097640156745911, 1.0037003755569458, 1.5251184701919556, 0.9638595581054688, -0.22029872238636017, -0.8841073513031006, -0.8132955431938171, -0.7440993785858154, 1.9247726202011108, 2.874746084213257, 1.0554016828536987, -1.7811704874038696, -0.07355114072561264, -0.09139273315668106, 1.7326624393463135, -0.6819257736206055, 0.5089922547340393, 0.5881808996200562, 1.478350281715393, 0.7662376761436462, -1.858710527420044, 0.33020487427711487, 0.6414572596549988, 0.18800508975982666, -1.490320086479187, 0.4336116909980774, 0.5295091271400452], [1.5086027383804321, 1.5856778621673584, -0.26875194907188416, 0.09818527102470398, -0.6832417249679565, 0.9879756569862366, 0.5482854247093201, 1.4945515394210815, 0.6263195276260376, -0.508192777633667, -1.9792730808258057, 0.765040934085846, -0.44985049962997437, -0.35841625928878784, -0.49978095293045044, 0.09121523052453995, 3.6411423683166504, 0.6802833080291748, 1.2452808618545532, -0.5860592126846313, -1.3336290121078491, -0.8491557240486145, -1.363883376121521, -0.07691802829504013, 0.6237313747406006, -1.3701319694519043, 0.062427032738924026, 0.45432573556900024, -0.04352238029241562, 1.4427132606506348, 1.6369651556015015, 0.7456671595573425, -1.2611963748931885, -0.6304544806480408, -0.7954556941986084, -0.3974488079547882, 2.8369503021240234, -0.16985653340816498, 0.7106648683547974, 0.8901251554489136, 0.8159104585647583, 0.7512219548225403, -0.5678830742835999, -1.7342417240142822, 0.10609152913093567, -0.8267722725868225, 0.577913761138916, 0.5044432282447815, 0.4121565520763397, -0.484762966632843, 1.0297813415527344, 1.0311795473098755, 0.6140903830528259, -0.6393294930458069, 2.1881585121154785, -1.056573510169983, -0.7630736231803894, -0.33045920729637146, -0.25634023547172546, 0.4033205211162567, 0.3172147572040558, -1.6243374347686768, 1.6329575777053833, -0.6971950531005859], [0.21148569881916046, 1.095389485359192, 2.247321605682373, 0.34120383858680725, 0.673516571521759, -1.470889687538147, -1.288710355758667, 2.237830400466919, -0.24483861029148102, -0.07060577720403671, 0.17605622112751007, 0.6842614412307739, 0.1064741387963295, 0.20721270143985748, 1.1014456748962402, 0.49139973521232605, -0.4496936798095703, 1.6171149015426636, 0.3651871979236603, 0.6493955850601196, 0.28147679567337036, -0.5733612179756165, 0.23146644234657288, 1.5007294416427612, -0.941820502281189, 0.8938075304031372, -0.30399414896965027, -0.10623639076948166, -0.17852111160755157, 0.5703567266464233, -0.1268686056137085, -1.1297037601470947, 0.25921809673309326, 1.9424632787704468, 0.8574175238609314, -1.0779377222061157, 0.2789123058319092, -1.1481152772903442, -0.48767149448394775, 2.287921905517578, 1.4779101610183716, -1.3258886337280273, -0.12404825538396835, 0.1623561829328537, 0.6590875387191772, 0.7385165095329285, 2.2565839290618896, 0.7558541893959045, 0.20899595320224762, 0.43448469042778015, 2.3840646743774414, -0.10618440061807632, 1.868361234664917, 2.914311170578003, -0.27747049927711487, 0.5543800592422485, 0.008236308582127094, 1.3658924102783203, 0.24481478333473206, 0.7914206385612488, 0.6152777671813965, -1.4755339622497559, 0.7397541999816895, 1.0399073362350464], [2.187896490097046, -0.3451600968837738, -0.9897716045379639, -0.06778199970722198, 1.7013428211212158, -2.5645787715911865, -1.421363115310669, 0.8882180452346802, -1.6027907133102417, 0.29944583773612976, 0.8809441328048706, -0.4385644197463989, 0.2038765251636505, -1.2456278800964355, 1.0324865579605103, -0.6277425289154053, -0.5105096697807312, -0.591536283493042, -3.7238404750823975, -0.9200787544250488, -0.18473230302333832, 0.7913512587547302, 0.2928023636341095, 0.6410892605781555, -0.9053001999855042, -0.3216840922832489, 1.4347971677780151, 1.2328646183013916, -1.1750617027282715, -0.4920025169849396, 1.6891098022460938, 0.9429099559783936, -0.6139597296714783, -1.6695220470428467, 0.4002993404865265, -0.36880525946617126, -0.027594376355409622, 0.9341369271278381, 1.0701287984848022, 1.869370460510254, -0.6190218925476074, 1.9397478103637695, -0.051502007991075516, 1.6772122383117676, 0.9332075119018555, 0.8131510019302368, -0.07140982151031494, 1.2913063764572144, -0.5486074090003967, -0.3842696249485016, 0.8387001156806946, -0.19040298461914062, -0.1640137881040573, -0.5896903276443481, 1.402822732925415, -0.4381128251552582, -0.666821300983429, 1.6732794046401978, -0.7672736644744873, 1.2077080011367798, -0.4742015302181244, -1.8540452718734741, 0.44270822405815125, -0.3146287202835083], [-1.3020602464675903, -0.9011216759681702, -0.4507075250148773, -0.01663251593708992, -0.37217026948928833, -0.03334619849920273, -0.46058371663093567, -0.1167629212141037, -0.08828515559434891, 0.06985055655241013, -0.04239414632320404, -0.49374914169311523, 0.2827896177768707, -2.098629951477051, 1.8944724798202515, 0.31003397703170776, -0.5592662692070007, -0.06509634852409363, 0.4052235782146454, 1.2684261798858643, -0.5436011552810669, 1.422674298286438, -1.2951704263687134, -1.6158145666122437, -0.6750008463859558, 0.17525441944599152, -0.5113018751144409, -1.1405359506607056, 0.5466045141220093, 0.056611381471157074, -1.0405855178833008, 0.03049112670123577, -0.1930338591337204, 0.025445614010095596, 0.5581070780754089, -0.037593163549900055, 0.9464249610900879, 0.7994019985198975, -0.05037865787744522, -0.4885302782058716, -1.0548595190048218, 1.30940842628479, 0.2393466830253601, -0.4350621700286865, 0.8630970120429993, -1.2819472551345825, 0.3937314450740814, 0.5496341586112976, -1.5681270360946655, -2.3347363471984863, 0.4088849425315857, -1.1710407733917236, -1.1785125732421875, 2.0250203609466553, 0.9338444471359253, -2.2629082202911377, -1.1697113513946533, 0.9003051519393921, -0.7274747490882874, -0.25330477952957153, -0.9077224135398865, -0.2695174217224121, -1.7869690656661987, 0.14412735402584076], [-0.4318922758102417, 1.4064580202102661, -0.5578296184539795, -0.5955326557159424, 0.35715562105178833, 0.34775862097740173, -0.18240436911582947, 0.8117572069168091, -0.7287776470184326, 0.5676709413528442, 2.111478090286255, -0.4489826261997223, -0.23582004010677338, -0.06932801008224487, 0.5392339825630188, -1.685858130455017, 1.2943836450576782, 1.5616142749786377, 0.5661691427230835, -0.2602933347225189, 0.5394705533981323, -1.4522117376327515, 1.279659390449524, -0.4424813389778137, -1.0858736038208008, -0.699604332447052, -0.40895241498947144, -1.1981797218322754, -1.6888819932937622, 1.0752480030059814, -0.2867329716682434, 0.3017469644546509, -1.0868562459945679, -0.27791792154312134, 1.9981212615966797, -1.3742486238479614, 0.3696935772895813, -0.34442025423049927, -1.7455037832260132, -0.30012863874435425, 0.6988290548324585, -1.7320449352264404, 2.630110740661621, 0.9505618810653687, 2.2768354415893555, 1.096788763999939, -1.1880050897598267, -0.20689663290977478, 0.4657042324542999, -1.750864863395691, -0.7483610510826111, -1.4681732654571533, 0.2736184298992157, 0.2721119523048401, 1.7869054079055786, 1.8449671268463135, -1.3302603960037231, 0.4009573459625244, -0.01483177486807108, -0.13801419734954834, -0.9935472011566162, 0.016724267974495888, -0.9866944551467896, 0.019896289333701134], [-1.4936805963516235, 0.8947629332542419, 1.4560328722000122, -1.392266035079956, -1.6964730024337769, -0.016209036111831665, 1.3721184730529785, -1.226966381072998, -0.739453136920929, -1.5730235576629639, 1.0929392576217651, -0.4003080725669861, 0.756984293460846, -0.4689331352710724, 0.17760039865970612, 1.3265564441680908, -0.4915331304073334, 0.9265952706336975, 0.5143378973007202, -0.12479972094297409, -0.5865947008132935, 1.4544298648834229, -0.20199766755104065, 0.39587438106536865, -0.6901803612709045, -0.45887938141822815, -0.7226492166519165, 0.9728906750679016, -0.9286075234413147, -1.559976577758789, -1.5713908672332764, 0.22763434052467346, 0.7529057264328003, 0.09001453965902328, -0.5451507568359375, -1.8315943479537964, -0.527711033821106, -0.5675747394561768, -0.8657614588737488, 1.2618529796600342, -0.5056012868881226, -0.4401618242263794, -0.03949524462223053, 0.05994359776377678, 2.05686616897583, 1.5641059875488281, -0.25514569878578186, -0.2570342719554901, 1.557645559310913, -0.13218432664871216, -0.6991535425186157, 0.6149659752845764, -1.1300814151763916, -0.0992775708436966, -1.0676376819610596, -1.1467839479446411, -0.8902455568313599, 0.9112922549247742, 2.973656177520752, -1.998653531074524, -0.4808400571346283, -0.0746035948395729, 0.5857413411140442, 1.444819450378418], [-0.8929635882377625, -1.9423205852508545, -0.7467467188835144, -0.8298897743225098, -0.692716658115387, 1.1886793375015259, -0.49848589301109314, -1.767353892326355, -0.2852246165275574, 0.1607421338558197, -1.108292579650879, -0.07149351388216019, 0.5084854364395142, -0.37429502606391907, 0.605114758014679, 0.38494443893432617, -0.9004049897193909, 2.1734468936920166, -0.3167034983634949, -0.4932340979576111, 0.5040085911750793, 0.03914128243923187, 0.1708400547504425, -0.8490387797355652, -0.012006636708974838, -0.29113930463790894, -0.17432080209255219, 2.3264126777648926, 1.323663353919983, 0.3566519618034363, -0.7835521697998047, 0.33795008063316345, -0.3143985867500305, -0.43887919187545776, 0.11911219358444214, -0.7385275959968567, 0.9244892001152039, 1.9813793897628784, 0.2501565217971802, 1.0302032232284546, -0.3811984956264496, -0.8822641372680664, -0.42709052562713623, 0.40034887194633484, 0.24403001368045807, 0.015665452927350998, 0.3405716121196747, -0.522122859954834, 0.8757718801498413, -0.7431963682174683, 1.3139619827270508, 0.10906248539686203, -0.6168901324272156, -1.2970267534255981, -0.4257802665233612, 0.5103033185005188, 0.23462603986263275, 0.2762164771556854, 0.9134629368782043, 0.8829576373100281, -0.1077427864074707, 0.3917980194091797, 0.4220314919948578, 1.7164770364761353], [0.10996782779693604, -0.7861489057540894, -1.0805625915527344, 0.7695388197898865, -1.025148868560791, 0.19752812385559082, -0.8975578546524048, -0.5983275175094604, -0.3975095748901367, -0.5745249390602112, -0.24227625131607056, -0.9207600355148315, 0.9995127320289612, -1.0403939485549927, -1.5321166515350342, -1.4151010513305664, -0.026731273159384727, 0.4768016040325165, -0.054495636373758316, -1.12066650390625, -1.0366729497909546, -0.14490757882595062, 0.707606852054596, -0.7534310221672058, 0.057373471558094025, -0.31852272152900696, -0.5710068345069885, -1.2964632511138916, 1.253305196762085, 1.3873498439788818, -1.5762906074523926, 0.9418116807937622, -0.7073776721954346, -1.3403103351593018, 0.9027871489524841, 0.6960449814796448, -1.774005651473999, -0.5128499865531921, 0.1433914303779602, 0.9266537427902222, -1.0511813163757324, 2.6021389961242676, 0.6075392961502075, 0.6664815545082092, -0.5504602789878845, 0.4944800138473511, -0.6744384765625, -1.0697369575500488, 0.1137169897556305, -0.48502615094184875, -1.3400146961212158, -0.256594717502594, 1.6301853656768799, -0.7065580487251282, -0.8943641185760498, -0.4910971224308014, -1.5654830932617188, 0.6488543748855591, 0.7520965933799744, -0.3792562484741211, 0.15539926290512085, 0.8577190041542053, -0.7522007822990417, -2.1719231605529785], [-2.1815147399902344, -0.10691674798727036, -0.36782804131507874, 0.27361518144607544, -0.3645760118961334, 0.9821302890777588, -1.325888991355896, -1.645438313484192, -0.8476234674453735, -1.07047700881958, -0.7081170082092285, -1.2824209928512573, 1.4751778841018677, -1.1760073900222778, 3.0972235202789307, -0.8562607765197754, 1.0567153692245483, 0.654826819896698, -1.0851659774780273, 1.4024368524551392, 1.1007511615753174, 0.04016612470149994, 1.6241360902786255, 0.3204520642757416, -1.3810622692108154, 0.47086024284362793, -0.3160563111305237, -0.03318158909678459, 0.13603101670742035, -1.1206532716751099, -0.6412895321846008, -0.16246022284030914, 0.8147621154785156, 2.1181302070617676, -0.9001829624176025, -0.9853687882423401, 0.48999401926994324, -0.07406897097826004, -1.448530912399292, -0.44271063804626465, 1.1532269716262817, 0.7065620422363281, 0.07197185605764389, 0.4552416205406189, 0.4882577359676361, -0.6929700374603271, -0.30593180656433105, 0.7027683854103088, 0.04486541077494621, -0.5811822414398193, -0.21153466403484344, 0.9434122443199158, 2.2469570636749268, 0.44937726855278015, -1.3000174760818481, -0.47100383043289185, -0.8418230414390564, 0.45015600323677063, 0.3370760679244995, 0.39772263169288635, -1.8849197626113892, -1.1406980752944946, 0.1389055699110031, -0.058801230043172836], [-0.7276543974876404, -1.2485132217407227, 1.8882700204849243, 0.06252133846282959, -0.4067169427871704, -1.417598009109497, 1.1462911367416382, 0.29914575815200806, 0.6864024996757507, -0.13037408888339996, -2.501641035079956, -1.7858830690383911, 0.059915341436862946, 1.2302660942077637, -1.283221960067749, -1.677004337310791, 0.5266416072845459, 1.0592286586761475, -1.2758020162582397, -0.6305095553398132, -1.1183549165725708, 1.4080138206481934, -0.2390836775302887, 1.089268684387207, -0.3489867150783539, 0.21244323253631592, 0.39615964889526367, -0.16967982053756714, 1.2845040559768677, -0.25607162714004517, 0.6375734806060791, -1.2408045530319214, 0.7378155589103699, 0.9677716493606567, 0.13315516710281372, -1.5971543788909912, -0.2781117558479309, 0.27045390009880066, 1.2357105016708374, 2.3902885913848877, -0.3576813340187073, -0.5867815613746643, 2.0763425827026367, -0.29186108708381653, -0.10555653274059296, 0.9184828400611877, -1.0646541118621826, -0.321556031703949, -0.39090973138809204, 0.23279672861099243, -0.5679233074188232, -0.35766103863716125, 0.5101493000984192, -0.6299359202384949, 1.4021211862564087, -0.15364733338356018, 1.593063235282898, 0.7701926827430725, -1.4700450897216797, -0.5337753891944885, -0.29688242077827454, -0.33107906579971313, 1.6424535512924194, 1.7688446044921875], [-1.055328607559204, -0.5436139106750488, -0.8422835469245911, 1.1032295227050781, 0.7087205648422241, 0.39325955510139465, -0.9902739524841309, -0.019783347845077515, -0.3237631022930145, 0.13984307646751404, 2.472668170928955, -1.2240859270095825, 0.33002811670303345, 0.009120040573179722, 0.7460440993309021, -3.013162136077881, -0.43009811639785767, -0.1028466671705246, -0.33838629722595215, 0.8485738635063171, -1.5070521831512451, -1.309998631477356, 1.1277798414230347, -0.6933311820030212, 0.8756696581840515, 1.6292282342910767, -0.5980071425437927, -0.5258448719978333, -0.41480687260627747, 0.8173183798789978, -1.1464297771453857, 0.6617013216018677, 0.5892597436904907, 1.5611261129379272, -0.8527030944824219, 1.452401041984558, -0.9957641363143921, 1.3182822465896606, 2.2641239166259766, -0.2752700448036194, -1.1275849342346191, 0.6335533261299133, -1.268418312072754, -0.05347198620438576, 0.013002193532884121, -0.8186050653457642, -0.7807636260986328, -0.4249461889266968, 0.4328417479991913, 2.6378514766693115, -0.668634831905365, 0.24279052019119263, 0.6255132555961609, -0.6091211438179016, 0.10202402621507645, -0.8642440438270569, -0.49785658717155457, -0.857238233089447, 0.5100051760673523, -0.4243707060813904, -0.43906843662261963, -1.067441463470459, -0.12787356972694397, 1.4357810020446777], [1.7925382852554321, -0.2694689631462097, 0.5712762475013733, 1.043411135673523, -0.3039679229259491, -1.3895734548568726, -0.9519476890563965, -1.2190732955932617, -0.8195492029190063, -0.4962933361530304, -0.23423726856708527, 0.4083276391029358, 0.3431546688079834, -1.4156190156936646, -2.194064140319824, -1.1001560688018799, 0.15972550213336945, 0.12110937386751175, -1.0810790061950684, 0.13236676156520844, 0.5191200971603394, -1.4844621419906616, 1.3136838674545288, 1.3107727766036987, -0.17854255437850952, -1.1459800004959106, -0.253866583108902, -0.5999504327774048, 0.08328220248222351, 0.09530922770500183, -0.34606653451919556, 2.432910919189453, 0.25740036368370056, -0.6483843922615051, -0.1438734233379364, -0.09647811204195023, -0.3384907841682434, 1.310962200164795, -0.5993762612342834, -0.3809540271759033, -0.7332083582878113, -1.621930718421936, -0.20943865180015564, -0.03508728742599487, 0.25092896819114685, -1.3997044563293457, 2.717331886291504, 0.5580447316169739, -0.3754560649394989, -1.1989166736602783, 0.022218924015760422, -0.09530811756849289, -0.3014928102493286, -0.5620977878570557, -0.012332255952060223, 1.1978869438171387, 0.3839755654335022, -0.6740188598632812, 0.60390305519104, 1.0666706562042236, 0.8832350373268127, -0.00030427947058342397, -0.7767381072044373, -1.5438971519470215], [-1.238323450088501, 0.1450534611940384, 0.7536243796348572, 0.743087649345398, 0.47502461075782776, 0.14841780066490173, -0.6326096057891846, -0.9020038843154907, 2.0923447608947754, 0.7146176695823669, -0.2883818447589874, -1.5778138637542725, -0.5588877201080322, 0.4892173707485199, 1.411101222038269, -0.4641875922679901, -0.42213931679725647, -0.40665799379348755, -0.4505403935909271, 0.7739818096160889, -1.8155946731567383, -0.9852719306945801, 0.10443665087223053, 0.6506636738777161, 0.1378553956747055, 0.4323311448097229, 0.387359619140625, 0.45455098152160645, 0.9107966423034668, 0.06653673946857452, 0.3694364130496979, 0.19989413022994995, 1.5093578100204468, -0.6384032964706421, -0.41525179147720337, -1.0103543996810913, 2.2945940494537354, -0.6584151983261108, -0.9570698142051697, -0.9461970925331116, 0.6155100464820862, -1.2541322708129883, -0.9445359706878662, -0.47416314482688904, -1.175646424293518, 1.0808677673339844, 1.0502532720565796, -0.7016475796699524, -0.5748838782310486, -0.8505844473838806, 0.5524767637252808, 0.5488824844360352, -1.0842475891113281, 0.010061308741569519, -0.5649890899658203, 1.0865617990493774, 0.47481802105903625, -0.29097622632980347, -0.5196343660354614, -2.5280961990356445, 0.03652729094028473, 0.14385682344436646, -0.6172986626625061, -0.5188766717910767], [1.938004732131958, -1.8442652225494385, 0.02957642264664173, -1.015297532081604, -2.3203110694885254, -1.1902294158935547, 1.955323576927185, -1.1935547590255737, 1.1814155578613281, -1.203141689300537, -0.28958725929260254, -0.06508639454841614, 0.7528735399246216, -0.2300168126821518, -0.7261694073677063, -0.7372326254844666, -0.730995237827301, -0.42098650336265564, 0.7713159322738647, 0.5066797137260437, 1.1925650835037231, 0.9646909236907959, 0.5600964426994324, 1.6178152561187744, -0.6848145723342896, 0.6671558618545532, 0.4212408661842346, -1.3347492218017578, -1.4418056011199951, 0.057089030742645264, -1.338630199432373, -0.6315028071403503, 0.008123681880533695, -1.260028600692749, 1.0362236499786377, 0.5351344347000122, 1.1323356628417969, -0.9117847681045532, -0.565697193145752, -0.04364938661456108, -1.4198969602584839, -1.874587893486023, 0.1505689173936844, 0.44060471653938293, 1.2908141613006592, -0.48418691754341125, -1.1600583791732788, -0.6377032399177551, 0.9854670763015747, -0.48462918400764465, 0.6374494433403015, 0.2575633227825165, -0.039611682295799255, -0.5667101144790649, -0.34898310899734497, -0.34604090452194214, -0.5821437835693359, -0.034920670092105865, 0.820773720741272, 1.7769732475280762, 0.4829919636249542, -0.03141709417104721, 2.8622312545776367, 1.9470311403274536], [1.3048020601272583, -0.04363364726305008, 0.14834927022457123, 0.9206578731536865, -0.022531863301992416, 1.9450827836990356, -0.46847254037857056, 0.3088984191417694, 1.2364592552185059, 0.3266957998275757, -0.6849087476730347, -0.5569748878479004, -0.4242161214351654, 0.7635106444358826, -0.12903593480587006, -0.2973252832889557, -0.28207874298095703, 0.946437418460846, -1.5950642824172974, -0.5081101059913635, 0.17519396543502808, -0.5442025661468506, 0.4313600957393646, 1.1580754518508911, 0.40851423144340515, -0.30854153633117676, 0.6832914352416992, -1.2503833770751953, 0.18556083738803864, -0.7679393291473389, -0.9090229272842407, 0.30195319652557373, -0.00742282485589385, 0.13914991915225983, -0.006066076457500458, -1.6134910583496094, 1.213732361793518, -0.8339917659759521, 0.34127506613731384, 1.2379505634307861, 0.668225109577179, 1.4130816459655762, 1.34465491771698, -0.2766619622707367, 0.1179363951086998, 0.6755293011665344, 1.1534684896469116, 0.6087758541107178, -0.09318558126688004, -0.5625423192977905, 1.2133853435516357, 0.8252180218696594, -0.7035451531410217, 1.5894441604614258, 0.11119166761636734, 1.2492622137069702, -0.1703796535730362, -0.6807985901832581, -2.28829026222229, -0.3333736062049866, -0.36142271757125854, -1.084536075592041, 0.38667935132980347, 0.2539348304271698], [-0.5885739922523499, 0.5535568594932556, -0.21634164452552795, 0.6533682346343994, 0.8144567608833313, -0.6138684749603271, 2.141204357147217, 1.1017507314682007, -0.311663419008255, -0.46656814217567444, 1.201856255531311, 1.263608455657959, -0.473611980676651, -0.8586840629577637, -0.8296266198158264, 0.025844819843769073, -0.9810721278190613, -3.1663389205932617, -0.972769558429718, -0.4472154974937439, -1.065438151359558, 0.8721407055854797, -0.9327455163002014, 0.8313166499137878, 0.9574812650680542, 0.5458251237869263, 0.3295494318008423, -0.47771716117858887, 0.32129672169685364, 1.2172075510025024, 0.4475543200969696, 1.339826226234436, 0.3411267101764679, 0.5860694646835327, -0.8689785003662109, 0.12762507796287537, 0.699853241443634, -1.2946332693099976, 2.5994508266448975, -0.35567113757133484, 0.4937787652015686, -0.9177879691123962, -0.3737757205963135, -0.411350280046463, 0.25989916920661926, 0.3878340423107147, 1.0562591552734375, -0.47500139474868774, -0.4850253462791443, 0.016458410769701004, -0.11610597372055054, 1.0757226943969727, -0.006933359894901514, -0.3076377809047699, 1.1079248189926147, -0.24074538052082062, -0.07578544318675995, -1.2365410327911377, 0.5324252843856812, 1.0117645263671875, 0.9604724049568176, 0.3353769779205322, 0.30676743388175964, -1.7848024368286133], [-0.8931236863136292, -0.06116329878568649, 0.6038745641708374, -0.33773308992385864, -0.18794147670269012, -0.47213518619537354, -1.2685531377792358, -1.1582967042922974, -0.5847799181938171, -0.9158594608306885, -1.8408176898956299, 0.5996549129486084, 1.3007277250289917, -0.7694937586784363, 0.13463646173477173, 1.0189456939697266, 0.6985918879508972, 0.9577822685241699, 0.34324854612350464, -0.8341532945632935, 1.7820103168487549, 1.662069320678711, -0.44346943497657776, -0.3726212978363037, 0.9068967700004578, -0.5043227672576904, 0.03462400659918785, 2.0758438110351562, -0.8672733902931213, -0.04669330269098282, -0.6457329392433167, -0.18621061742305756, 0.617798388004303, 0.7687378525733948, -0.6802349090576172, 1.3941352367401123, 0.06635450571775436, 0.15087731182575226, -1.0030690431594849, 1.013624668121338, -2.047823190689087, 0.7927367091178894, 0.36255383491516113, -0.18407197296619415, 0.31824639439582825, 0.6290716528892517, -1.0911809206008911, 0.19527123868465424, -1.2117401361465454, -0.8895631432533264, 2.074056386947632, 1.345262885093689, 0.16369709372520447, -1.2607907056808472, 2.157353162765503, 1.4733306169509888, -0.9957746863365173, -0.27755624055862427, -0.061437711119651794, -0.5665071606636047, 0.1819145828485489, -0.9188873767852783, -0.8887435793876648, 0.505330502986908], [-1.3991737365722656, -0.3544650077819824, -0.913382351398468, 0.45975252985954285, 0.3811306059360504, -2.4507083892822266, 0.5390725135803223, -0.39641621708869934, 0.3748718202114105, -0.869085967540741, -0.6597225069999695, 0.9240934252738953, -1.4135634899139404, -2.419215202331543, 1.101980209350586, -0.023768028244376183, 1.2008010149002075, -0.45305654406547546, -0.49797287583351135, 0.7055266499519348, -1.3137075901031494, -1.578776240348816, 1.3417435884475708, -0.6559526324272156, -1.14935302734375, 0.06037121266126633, -1.0889179706573486, 0.1379016637802124, 0.33554425835609436, 0.2701999247074127, -0.13412117958068848, 0.285540908575058, 0.7608290314674377, -0.39817196130752563, 1.155226230621338, 0.8763236403465271, -0.5023056864738464, -0.7875507473945618, -2.5930585861206055, -1.2786674499511719, -0.31266823410987854, 1.514684796333313, -1.7171484231948853, -1.3297193050384521, -1.1875249147415161, -0.8386409878730774, 1.6771798133850098, -0.13378271460533142, -0.009793034754693508, -1.0434359312057495, -0.4617554545402527, 1.2490336894989014, 0.4589407444000244, 0.010633355006575584, -0.5393967628479004, 1.4267860651016235, 0.2109602838754654, 2.018669366836548, 0.002670970745384693, -1.2542400360107422, 0.7344024777412415, -0.46651753783226013, 0.544853925704956, 1.6091681718826294], [-1.7039039134979248, -0.07422653585672379, 0.705990731716156, -1.0687108039855957, 1.007641077041626, 0.038123343139886856, -2.3134775161743164, 0.8405919075012207, 0.0697021484375, -1.1213914155960083, 0.32864218950271606, -0.5917321443557739, -1.0452558994293213, -0.7528963088989258, 0.1450636088848114, 1.816389560699463, 0.1876155436038971, -0.9832796454429626, 0.49520671367645264, -0.19010025262832642, 0.4808199405670166, 0.04366724565625191, 0.6132384538650513, 0.19971337914466858, -1.2431068420410156, 1.223439335823059, -1.3107633590698242, -0.4446941018104553, 0.16134124994277954, 1.0531684160232544, 1.276293158531189, -0.20672908425331116, -0.26808181405067444, -0.815055251121521, 0.9215173125267029, 0.20697739720344543, 0.6310358047485352, -0.15389694273471832, -0.3089686632156372, 0.24328093230724335, -0.6686445474624634, 0.8207737803459167, -0.13668310642242432, -0.32243311405181885, -1.5169883966445923, -0.9986160397529602, -0.379141241312027, 0.5345569252967834, -0.15209458768367767, -0.9286942481994629, -0.7476134300231934, 1.2494540214538574, -1.3689074516296387, 0.5018811225891113, -0.7179873585700989, -1.404873013496399, -0.08814305067062378, 0.03800246864557266, -0.5131597518920898, -1.0918424129486084, 0.02341708354651928, 0.07928908616304398, 1.216223120689392, 0.4235437214374542], [0.16267114877700806, -1.4340009689331055, 1.3683674335479736, -1.490297555923462, -1.0563808679580688, 0.19019079208374023, 0.47402137517929077, 0.06513935327529907, -0.2953328788280487, 1.6432151794433594, 0.004360359627753496, 0.29266422986984253, -1.019966959953308, 1.8886770009994507, -0.6294687390327454, -0.5355812311172485, -1.0898594856262207, -0.09031964093446732, 0.7685688734054565, -0.16682963073253632, 1.2665210962295532, 1.2135915756225586, 1.154129981994629, 0.20419341325759888, 0.7598168253898621, 1.1724348068237305, 0.6664116978645325, -0.14618469774723053, 1.4176090955734253, -0.5502665638923645, -0.05813531205058098, 0.5434710383415222, 0.3661821484565735, 0.40749260783195496, 0.8882173895835876, 0.15619514882564545, -1.0036817789077759, 0.3548837900161743, 0.5475743412971497, -0.36886411905288696, 2.0012454986572266, -1.8080615997314453, -2.2926299571990967, -0.1410788893699646, 0.20153668522834778, -2.0060062408447266, 0.9639929533004761, -0.9115892052650452, 0.21865418553352356, -0.9747732281684875, -0.7958159446716309, 0.8730318546295166, -0.6557731628417969, -0.5192862153053284, 1.2030432224273682, -2.207045316696167, 0.739119291305542, -1.1505385637283325, 0.7391828298568726, 0.07940841466188431, 0.3795050382614136, -1.3979064226150513, -0.38221046328544617, -0.036770667880773544], [0.08640535920858383, 0.6510724425315857, -0.12352853268384933, -0.3249364495277405, 1.4084964990615845, 1.3846335411071777, -0.8465520143508911, -0.02114170603454113, 0.12273194640874863, 0.6349299550056458, -0.5940006375312805, -0.23453016579151154, -0.38705506920814514, -0.18907497823238373, -0.2887004315853119, -0.09731590002775192, 0.7708813548088074, -0.41215696930885315, 0.7220350503921509, -2.0677428245544434, 0.3897812068462372, 0.6053696274757385, 0.38846492767333984, -0.9796082973480225, 0.08109021931886673, 1.4319422245025635, -0.7239851951599121, 0.49152350425720215, 0.8050404787063599, 0.45822224020957947, 1.3154982328414917, -0.8237961530685425, -0.4036748707294464, 1.0717556476593018, -0.42455190420150757, -0.22825615108013153, 0.9829425811767578, 1.1817095279693604, -1.4399689435958862, 0.14089666306972504, 2.1291451454162598, -0.9849526286125183, -1.5103353261947632, -0.37593597173690796, 1.0449458360671997, 0.45163601636886597, -1.044604778289795, -1.165249228477478, 1.211909532546997, -0.8604204654693604, 1.1067122220993042, 0.7742653489112854, -0.8132317662239075, -0.8129803538322449, 0.35809144377708435, -1.2662022113800049, -1.1112104654312134, -0.34255337715148926, -2.3567187786102295, 1.9263004064559937, 0.14010784029960632, -0.4328553378582001, -0.22679071128368378, 0.5761272311210632], [-1.7864668369293213, -0.07110567390918732, 0.6782741546630859, -1.5027596950531006, 1.5061172246932983, -1.4399698972702026, 0.5104739665985107, -0.9035555124282837, 0.279297411441803, 0.180782288312912, -0.1334613859653473, 0.8246036171913147, -0.10821675509214401, -1.0659250020980835, -0.632331132888794, 0.4826032817363739, -0.5517854690551758, -1.6943799257278442, -0.021722320467233658, -0.462631493806839, 1.4628286361694336, 0.19059786200523376, 0.09588494151830673, -2.107029438018799, -0.3943837583065033, 0.5622603297233582, 2.1436002254486084, 0.7428898811340332, 1.037050724029541, 0.8380247354507446, 0.3808043599128723, -1.023363709449768, -1.0310462713241577, 0.03495952859520912, -0.2681104242801666, 1.1077337265014648, 0.5501528978347778, 0.7489566206932068, 0.6597554683685303, 1.911105990409851, -0.2133493274450302, -0.22864334285259247, -0.9474093317985535, 0.29275983572006226, -0.3860778510570526, 0.5612918138504028, 0.6912298798561096, 1.0046077966690063, -0.4425083100795746, 0.7370120882987976, 1.4881157875061035, -0.1193515807390213, -0.43070587515830994, 0.7717206478118896, 0.24664762616157532, 0.6827230453491211, -0.8071386814117432, 0.1501459926366806, 1.1321985721588135, -1.914315104484558, 0.37144988775253296, 0.3606768548488617, 1.5667890310287476, -0.080202616751194], [-0.25870612263679504, -0.12046775966882706, 0.35268568992614746, -0.755757212638855, -1.0583302974700928, -0.3518233597278595, 0.97806316614151, 0.8290836811065674, 0.6713173389434814, -1.0877482891082764, 0.5777125358581543, -0.48910850286483765, -0.8415058255195618, -0.10551092773675919, -1.310378909111023, 1.5498234033584595, -0.4198298156261444, 0.7560045719146729, -1.1133639812469482, 0.6964901685714722, 0.060657259076833725, 0.2707692086696625, 1.0622934103012085, 0.9523278474807739, 0.5834928154945374, 0.9753918647766113, -2.392390251159668, 0.006042996887117624, -1.0819132328033447, 0.9008791446685791, 0.10518866777420044, 0.8040409684181213, 1.3473100662231445, 0.1982462853193283, 0.36784493923187256, 1.795417070388794, 0.7308107018470764, -0.4798928499221802, -1.1753214597702026, -2.059309720993042, 0.3525371253490448, -1.9145196676254272, 0.03138750419020653, 0.3621789515018463, -0.495606929063797, 1.541958212852478, -0.1701001524925232, 1.4025275707244873, 0.6938651204109192, 0.36941292881965637, -0.1466244012117386, 0.2871614396572113, 0.2864047884941101, 1.7160624265670776, -0.5490472912788391, -1.5991255044937134, -0.28599879145622253, -0.3598797023296356, -0.7535080313682556, -0.12558820843696594, -0.14388184249401093, 0.2639496326446533, 1.0802853107452393, 0.8424332141876221], [0.5168085694313049, 1.2472546100616455, 0.1426684558391571, -0.16898803412914276, -0.505683958530426, -1.5424789190292358, 0.5887784361839294, -0.564041018486023, -0.6875666379928589, -0.5772505402565002, -2.267848253250122, -0.845068633556366, -0.1807728409767151, 0.2304498851299286, 0.1586007922887802, -0.41027629375457764, -1.5360678434371948, 0.36198335886001587, -0.44181084632873535, -1.1805320978164673, 0.3383803069591522, -0.5497133135795593, -0.23132184147834778, 1.0192267894744873, 0.7517140507698059, -0.6908643245697021, 1.1949366331100464, -0.5313144326210022, -0.32169920206069946, -1.1423875093460083, -0.4479610025882721, 0.8458111882209778, 0.8978509902954102, 0.5436022877693176, -0.3421418070793152, 0.1262310892343521, -0.46088066697120667, 0.4091283679008484, 1.7479965686798096, 0.2805405855178833, 1.703655481338501, 0.7831084728240967, 0.04171939566731453, -1.855342984199524, -0.054447442293167114, 0.6493946313858032, -0.3715062439441681, -0.508931577205658, 1.271522879600525, 1.1312923431396484, 0.16735132038593292, -1.099053978919983, 0.8144210577011108, -1.678168773651123, 0.14163662493228912, 1.5890257358551025, -0.12841978669166565, -0.07132935523986816, -0.9180492162704468, 0.3338193893432617, 0.513879656791687, 0.5048772692680359, -1.3988560438156128, -0.3938349485397339], [-0.12338388711214066, 0.9330510497093201, -0.17160597443580627, 1.5758448839187622, 1.3248990774154663, 1.392491340637207, 0.9187530279159546, 0.6259981989860535, -1.430508017539978, 0.25713521242141724, -2.0048158168792725, -0.47645536065101624, -0.6736746430397034, 1.6097792387008667, -0.3153809905052185, 1.6393852233886719, 0.32967445254325867, -0.4114989936351776, -0.36531683802604675, 0.2880539894104004, 0.7478496432304382, 1.862586259841919, -0.4679616093635559, -1.546236276626587, 1.9209320545196533, -1.6905993223190308, 0.56766676902771, -0.7073571681976318, -0.07731334865093231, 0.4088427424430847, -0.9752285480499268, -0.4435969591140747, -1.7881525754928589, 0.8638768196105957, 0.0037573608569800854, -0.5079019665718079, 0.41068872809410095, -0.6407657861709595, 0.612634539604187, -0.5179256200790405, -0.7550125122070312, 0.72857666015625, -0.006232752464711666, 0.8306741714477539, -0.11903277039527893, -0.2369605153799057, 0.2070017158985138, -1.340543508529663, 1.1742355823516846, -2.553924798965454, 0.29011884331703186, -0.5023745894432068, -0.46865883469581604, 0.9641053676605225, -0.07671237736940384, -0.3484383523464203, 1.0516775846481323, -0.769271731376648, 0.457128643989563, 0.03473333641886711, -1.4179631471633911, -0.4624619483947754, -1.2206579446792603, 1.5535961389541626], [-0.7536552548408508, -1.2647846937179565, -0.4201355278491974, 0.03643021732568741, 1.2231578826904297, 0.163648784160614, 0.2872721552848816, 0.09885983914136887, 0.6843934059143066, 0.41439247131347656, 0.848422110080719, -0.010364530608057976, 0.6452234983444214, -1.8217874765396118, -1.0838088989257812, 0.22439903020858765, 1.3649102449417114, -1.0552228689193726, 0.3668697774410248, -0.6720731258392334, 0.347621887922287, -0.8916301131248474, -0.02386719360947609, 0.12450294196605682, -0.19761699438095093, -1.2865023612976074, -0.839931845664978, -0.08965576440095901, 0.815190851688385, 1.8145018815994263, -1.061944842338562, 1.0375670194625854, 0.9594526290893555, 0.014744177460670471, 1.329069972038269, -0.16409192979335785, 0.4134064018726349, -1.062698245048523, 1.3559662103652954, -1.0549988746643066, 1.2432955503463745, 0.7781395316123962, -0.2641839385032654, 0.5210358500480652, -0.29483723640441895, -0.07797744125127792, 0.6529724597930908, 0.10216688364744186, 0.1675157994031906, -0.20587660372257233, 0.396482914686203, 0.48485860228538513, 1.1758153438568115, -0.7179849743843079, 0.485841304063797, -0.8162551522254944, 0.3776935935020447, 0.09037400037050247, -0.6804623007774353, -0.5370046496391296, 0.3518866300582886, -0.45042699575424194, -0.5233562588691711, 0.7183871865272522], [0.6131876707077026, 1.2302231788635254, 0.21916744112968445, -0.019111808389425278, 0.2579791247844696, -0.46949249505996704, 1.8180524110794067, -1.2018815279006958, 0.027447814121842384, 1.2132915258407593, -0.03352905064821243, 0.514336347579956, 1.3073015213012695, -1.9333996772766113, -0.3365767002105713, 0.006669756956398487, -2.8558061122894287, 1.56210196018219, -0.9395695924758911, -0.6692719459533691, 0.024578893557190895, 1.0249104499816895, 2.5527803897857666, -1.3349734544754028, -0.9035511016845703, 0.06476447731256485, 0.5447749495506287, 0.03883512690663338, -0.4138619303703308, -0.10634186863899231, -1.0294358730316162, 0.41474536061286926, 0.21593593060970306, -0.6764122843742371, 0.757844090461731, 0.7641664743423462, -0.7382790446281433, 1.7855017185211182, 0.5469802618026733, 0.12262248247861862, -0.3308470845222473, 0.75994473695755, 0.5453864336013794, -1.7718538045883179, -2.4542124271392822, 0.2808830440044403, 0.9000207781791687, -0.6518194675445557, -1.1032779216766357, 1.1044946908950806, 0.5048139095306396, -0.11600425094366074, -0.8779472708702087, -1.0638427734375, 0.4893993139266968, 1.2675402164459229, 0.10024656355381012, -1.1869574785232544, -0.5448238849639893, 1.9287704229354858, 0.4364861845970154, 0.02793141081929207, 0.0912775844335556, 0.1884755790233612], [0.22447599470615387, 0.2997322976589203, -1.7005352973937988, 1.5719666481018066, 0.5255168080329895, 0.12432261556386948, 0.7856131792068481, -0.39087727665901184, 1.3873121738433838, 0.178683802485466, -0.3722492754459381, 0.07916080951690674, 0.7057914733886719, -0.5337357521057129, -0.5538418292999268, -0.7840363383293152, 0.22668734192848206, -1.928144097328186, 1.2542953491210938, -2.296431541442871, -0.23393285274505615, 1.3196772336959839, -0.6348210573196411, -1.482783317565918, 0.5545958876609802, 1.3473647832870483, 0.43150007724761963, 1.418914794921875, 0.37335753440856934, 0.1243230402469635, 0.4138911962509155, 1.6085842847824097, 0.5678942799568176, -0.39381247758865356, 0.8081120848655701, -0.7721688151359558, -0.44484490156173706, 1.2502005100250244, -0.2295212298631668, 0.24402660131454468, -0.3837682604789734, -1.021301031112671, -0.0923515111207962, 0.05483856797218323, -0.531352698802948, -0.2137196809053421, 0.7848465442657471, 0.06001770496368408, 0.45466580986976624, -0.007435949053615332, 2.0040769577026367, -0.38117021322250366, -0.3454779386520386, -0.660826563835144, -1.0931984186172485, 0.6100993752479553, -0.16027238965034485, -0.22337593138217926, 0.18431946635246277, 0.7162780165672302, 0.21838609874248505, -0.6483165621757507, -0.8189961314201355, -1.003555417060852], [1.126873254776001, 0.48239654302597046, 0.5237193703651428, 1.948892593383789, -0.025437047705054283, 1.0145446062088013, 0.3201039433479309, 1.7435133457183838, 0.7417260408401489, 0.017107708379626274, 1.4021140336990356, 1.234985113143921, -0.777003824710846, 1.4414100646972656, 1.6605523824691772, -0.5475636720657349, 0.3952140808105469, 0.2121819257736206, 0.1435362547636032, -1.7544726133346558, 1.086758017539978, 0.6894856691360474, 0.5369377732276917, -0.34834763407707214, 0.6704769134521484, -0.5202164649963379, 1.3519964218139648, -0.06762278825044632, -1.9329336881637573, -0.05685882270336151, 1.2556406259536743, 0.8761434555053711, -0.21353133022785187, 0.22623533010482788, 0.9151086211204529, 0.6613494157791138, 1.2021692991256714, 0.3435981273651123, -0.2495317906141281, -0.2571076452732086, -0.09483157843351364, 0.8049405217170715, 1.0391340255737305, -0.6202149987220764, 0.13477149605751038, 0.6546064019203186, 0.1756550371646881, 0.8228546380996704, -1.4429004192352295, 1.0380825996398926, -0.655663251876831, 0.4133659303188324, 0.863730251789093, 0.4682057201862335, -1.07380211353302, 0.014792470261454582, 0.4898635447025299, 0.5356130599975586, -1.863594651222229, 0.014951540157198906, 1.4494444131851196, -0.43632635474205017, -2.077580451965332, 1.2456914186477661], [-0.21190379559993744, 0.2191760241985321, -0.6266401410102844, 0.5535244941711426, -0.6017420291900635, 0.2915167808532715, 0.6639245748519897, 0.3421725034713745, -0.5676085352897644, -0.24491913616657257, 0.25517603754997253, 0.1087663546204567, -0.372962087392807, 0.21072715520858765, 0.06717436015605927, 0.5290808081626892, 0.7313655614852905, 0.9045648574829102, 0.7820014357566833, 0.7901997566223145, 0.08312267810106277, -0.4539681077003479, 0.3444499373435974, -0.055772025138139725, 1.294927954673767, -0.08558964729309082, -0.528766393661499, -1.0300219058990479, 1.8184791803359985, -0.46587011218070984, -0.9799431562423706, 0.5489470958709717, 0.1960759311914444, -1.2851901054382324, 1.301030158996582, 0.8022599816322327, -1.1545753479003906, -0.14017409086227417, -0.1147860437631607, -0.7887639999389648, -1.7831708192825317, -0.016884954646229744, 0.6828067898750305, 1.0709165334701538, -0.8567035794258118, -1.0964313745498657, -1.4764231443405151, -0.14303749799728394, 2.0427396297454834, 0.20910093188285828, -0.22698478400707245, 0.6882326602935791, -0.5256011486053467, -0.13990828394889832, 0.9235613346099854, -0.4107271134853363, -0.20506551861763, -0.865782618522644, -1.7074288129806519, -0.8183048367500305, 0.29213541746139526, 0.4213330149650574, 0.6956999897956848, -0.43218323588371277], [-1.933307409286499, -1.1333521604537964, -0.5911556482315063, -1.6767489910125732, -0.36841684579849243, -2.096937417984009, -0.30774733424186707, 0.42794373631477356, 1.5829278230667114, -0.7117906212806702, -0.3690211772918701, -0.7907461524009705, 0.038065411150455475, -1.2971465587615967, 0.24598011374473572, -1.6524425745010376, 0.5305007100105286, 2.084794759750366, 0.0015968598891049623, -1.253092885017395, -0.9296756982803345, -0.4594663977622986, -0.05755707621574402, -0.9069743752479553, 0.5945339202880859, -1.091690182685852, -1.6395624876022339, -0.24697691202163696, -1.6737782955169678, 0.10924182832241058, -2.545560836791992, -0.02477644570171833, -1.1195111274719238, 0.7523130774497986, 0.4630989134311676, 0.469835489988327, -2.058242082595825, -0.7309247851371765, 0.021525194868445396, 0.9393662214279175, 1.2957488298416138, -0.758950412273407, -1.194008708000183, -0.2706700265407562, 0.7461026906967163, 1.2447608709335327, 0.30577853322029114, 0.7582776546478271, 0.8350105285644531, -0.45385241508483887, -0.45341813564300537, 0.5116516351699829, -0.35405510663986206, 0.3660000264644623, -0.20623870193958282, -1.2917494773864746, 0.7570330500602722, 0.2381424456834793, 0.99125736951828, -1.6654928922653198, 0.04366661608219147, 0.005219210870563984, 0.06376242637634277, 0.023614821955561638], [1.178783893585205, -0.539273738861084, 0.3588131070137024, 0.2416352480649948, 1.3986756801605225, -0.27791085839271545, -0.2085413932800293, -1.9598255157470703, 1.0942561626434326, 0.0799676701426506, 0.47586947679519653, -0.9734147787094116, -0.8963679075241089, 1.2247422933578491, 0.6973381042480469, -0.3191170394420624, 1.527454137802124, -2.0296630859375, -0.9096241593360901, 0.7904620170593262, -0.21746304631233215, 1.1930387020111084, -1.4326980113983154, -0.574623703956604, 0.28290534019470215, -0.8447907567024231, -0.7848019599914551, 1.033366322517395, 0.8054175972938538, 0.5387402772903442, 1.3114882707595825, -0.89281165599823, 0.5226826667785645, 1.2441308498382568, -1.8798116445541382, -0.6500604152679443, -0.3757815659046173, -0.6284930109977722, 0.5962216258049011, -0.16439755260944366, 0.7401658296585083, -0.8062672019004822, 0.2612563967704773, -0.7812319993972778, -1.22859787940979, 0.9110577702522278, -1.5139288902282715, -0.6788738369941711, -0.5865289568901062, -2.2181291580200195, 0.08609369397163391, 0.7283382415771484, 0.3249531388282776, -1.7332614660263062, -0.1756853610277176, -1.1825026273727417, 0.6743296980857849, -1.2128146886825562, 0.7849063873291016, -0.3865047097206116, 3.495906352996826, -0.009648664854466915, 0.8236125707626343, 0.8699784278869629], [-2.174553155899048, -0.2593670189380646, 1.9771698713302612, -0.3181939125061035, -1.4917707443237305, 1.438031554222107, 0.9962617754936218, -1.0615137815475464, -0.6798891425132751, -0.38742929697036743, 1.6777751445770264, 0.7381132245063782, 1.3653844594955444, -0.5836111903190613, 0.18668721616268158, 1.0731604099273682, 0.5431594252586365, 0.5051502585411072, 0.7745699882507324, 0.3378649055957794, 0.41852203011512756, 0.891169548034668, -0.9725949764251709, 3.0975351333618164, -0.49915823340415955, 1.5343890190124512, -1.4267810583114624, -0.9485030174255371, -0.10983762890100479, -0.6408174633979797, 1.7921539545059204, -0.7705476880073547, -0.7295897603034973, -1.7651169300079346, 0.5999254584312439, 0.2859329879283905, -0.9493371248245239, -1.5889439582824707, 0.9224647879600525, -2.4541916847229004, 1.1786348819732666, 1.3729305267333984, 1.2440365552902222, 0.34334927797317505, -0.850106954574585, -1.2612496614456177, -0.1284337341785431, 0.06408105045557022, -1.2584468126296997, -1.3213977813720703, 0.5244626998901367, 0.7353151440620422, -1.3875977993011475, 1.9280648231506348, 0.7052067518234253, 1.9342429637908936, -0.9969397187232971, -0.3282700777053833, 0.07007167488336563, -0.030883830040693283, 0.2548139691352844, 1.1712110042572021, 0.18333394825458527, -0.9160425662994385], [-0.4346621334552765, 0.16188719868659973, -0.08586543798446655, -0.6670160293579102, -0.5071178674697876, 0.9157369136810303, 0.5100668668746948, 0.59587562084198, -0.9079983234405518, -0.7722702622413635, -0.6179609298706055, -1.2598400115966797, -0.7288339734077454, 0.36139345169067383, -0.7693425416946411, -1.0590604543685913, -0.9639296531677246, 0.311383992433548, -0.598160445690155, -1.7428001165390015, 0.9653606414794922, 0.10318035632371902, -0.7948008179664612, -0.5411607623100281, 0.7899106740951538, 1.1645983457565308, -0.4621693789958954, 0.34231194853782654, -0.1932712197303772, 2.0526180267333984, 0.9985488653182983, -1.7956944704055786, 0.22832566499710083, -0.7033340930938721, 1.763776421546936, -0.040131986141204834, 0.05209193378686905, -0.7579561471939087, -0.8914008736610413, -1.6178712844848633, 1.541425108909607, -0.287830650806427, -1.3523503541946411, 0.1568715125322342, -0.04123876616358757, 1.6459627151489258, -0.9128415584564209, 0.6797806620597839, -0.8902620673179626, -0.09772653877735138, -0.6346443891525269, -2.159703493118286, 0.4571278691291809, 1.411089539527893, -0.47780781984329224, 1.329129934310913, -0.12292880564928055, -0.2788461446762085, 0.8195744156837463, -2.468031883239746, -0.9073261022567749, 0.510093092918396, 1.6402928829193115, -0.25816255807876587], [-0.7852256894111633, -1.1965184211730957, -1.2141685485839844, -1.849709153175354, -0.4751931130886078, -1.3486640453338623, 0.6180236339569092, -0.031253475695848465, -0.07107633352279663, 0.12273119390010834, 0.9835339784622192, -1.650086522102356, -0.9548491835594177, 0.5185906887054443, -1.1004090309143066, -0.356229692697525, 1.2960879802703857, -0.09524605423212051, 0.3425953686237335, 1.052844762802124, -0.717117965221405, -0.6402711868286133, 0.5929355025291443, 0.06579708307981491, 0.46600961685180664, -1.6301130056381226, -1.8680177927017212, -0.7163955569267273, 1.4679172039031982, 1.0154199600219727, 0.2259567826986313, -0.06309661269187927, -1.1510417461395264, 0.6834554672241211, -0.41790345311164856, -0.6331480741500854, -1.8508065938949585, -1.1108720302581787, -2.1678802967071533, 0.14519478380680084, 0.15178057551383972, -0.5501481294631958, -0.2051997184753418, 0.5104585289955139, 0.7387785911560059, -0.926334798336029, 0.6626251935958862, -0.2306867241859436, 0.14458414912223816, 1.229239583015442, 1.369815707206726, -1.048558235168457, -1.2849680185317993, 0.7027066946029663, 1.640276312828064, -0.027092650532722473, -0.46494680643081665, -1.7247391939163208, -0.3793923556804657, 0.47959062457084656, 1.5948848724365234, 0.722893476486206, -0.01648665778338909, -2.0627689361572266], [0.11249858886003494, -0.6954333186149597, -1.900879979133606, 0.6519216895103455, 0.8219353556632996, -0.4520324766635895, -1.1050803661346436, -0.33840614557266235, -0.09150805324316025, 0.16547732055187225, -1.8079882860183716, -0.9078790545463562, 2.604675054550171, -0.8387749195098877, -0.9220901727676392, -1.9174566268920898, 1.057518482208252, 1.1330753564834595, -0.6584224104881287, -1.9359543323516846, -0.9193555116653442, -0.3855725824832916, -1.2311632633209229, -0.38185837864875793, -0.1851411759853363, 1.0008946657180786, 0.0010350878583267331, 1.0394161939620972, 0.1574210226535797, 1.623551845550537, -0.42156416177749634, 1.6283442974090576, 0.38195353746414185, -0.34199827909469604, 0.8340292572975159, -1.5782904624938965, -1.2903140783309937, -0.5306954383850098, 0.052766598761081696, 0.04415632784366608, -1.1538013219833374, 1.429674744606018, -0.9924612045288086, -0.36299267411231995, 0.45162898302078247, -0.5813034176826477, 1.0743153095245361, 0.559618353843689, 0.10232840478420258, -0.9318245053291321, 0.9018454551696777, 0.9052469730377197, -1.6109871864318848, -1.148161768913269, 0.08573154360055923, -1.2298694849014282, -1.943995475769043, 0.898337721824646, 0.5904995799064636, 0.08829983323812485, -1.4812175035476685, -1.764764428138733, -0.8552833199501038, -0.787409245967865], [-0.5908180475234985, -1.3782891035079956, -0.040446937084198, 0.6458027958869934, -0.27593451738357544, 1.6075952053070068, 0.08110252767801285, -0.98593670129776, -0.7486839890480042, -0.5641018748283386, -1.770541787147522, -0.5113611221313477, -1.2057974338531494, -2.928166389465332, 0.9014951586723328, 0.060624655336141586, 0.169235959649086, 0.030832499265670776, -0.3137476146221161, 0.15567646920681, 1.5661320686340332, 0.3887072801589966, -0.7939162254333496, -2.537771224975586, 1.4675488471984863, -0.38375845551490784, 0.3280465304851532, 0.8742849230766296, 0.3669593930244446, 0.09962274879217148, -0.6603330373764038, -0.6909527778625488, -0.32709914445877075, -1.9200186729431152, -0.3277016282081604, 2.2309560775756836, 0.17471875250339508, -1.9472352266311646, -0.32993021607398987, -0.5266562104225159, 0.9381444454193115, -0.8665686249732971, 1.0464246273040771, -1.8370428085327148, -1.2808960676193237, 0.33771663904190063, 1.3087692260742188, 0.3932048976421356, -0.8988767266273499, 0.23695839941501617, 1.6432673931121826, -0.9852432608604431, -0.08646494150161743, 0.2828931212425232, -0.3137590289115906, 0.7890336513519287, -1.4887564182281494, 0.8384009003639221, 0.29594001173973083, -0.5349651575088501, 0.9459143280982971, 0.25397196412086487, 0.35727107524871826, 0.7330767512321472], [0.7694299817085266, 1.255381464958191, 0.677756667137146, -1.1355739831924438, 1.8795207738876343, 0.9296957850456238, 2.212707281112671, 1.858466625213623, 0.06857142597436905, -0.8332356810569763, 1.039028286933899, -0.13967114686965942, -0.08333607763051987, 0.44038712978363037, 0.9797820448875427, 0.18117542564868927, -0.11922618746757507, -0.10947395861148834, -1.2055026292800903, 0.3384404182434082, -0.11450258642435074, -0.7443907260894775, 1.7395107746124268, -2.448209285736084, -1.4383292198181152, -0.9083642363548279, -0.3881925642490387, -1.1341300010681152, -0.4998837113380432, -0.7535348534584045, -0.37521082162857056, -0.4122961163520813, 0.19012688100337982, -1.028222918510437, 0.15450021624565125, -0.6779072284698486, 1.0390268564224243, -0.6851742267608643, -0.8883583545684814, 0.26456671953201294, 1.1555863618850708, 0.2577304244041443, 0.13935396075248718, -1.52734375, -1.1069144010543823, -0.12190170586109161, 2.509392023086548, 0.6582477688789368, 0.16266483068466187, 0.27409741282463074, -0.8833891749382019, 0.027682365849614143, -1.418278694152832, 0.14938309788703918, 0.21675361692905426, -0.9499405026435852, -0.7049860954284668, -0.9396502375602722, 0.7191416621208191, 0.882917046546936, 1.50018310546875, 0.6194955110549927, 0.265665203332901, -0.6960739493370056], [-0.19187459349632263, 1.094768762588501, -1.8162107467651367, 1.7060900926589966, 0.18394362926483154, -0.37624019384384155, -1.8578541278839111, -2.348839521408081, -1.2918034791946411, -2.576608657836914, -0.06731748580932617, 1.9050514698028564, -0.15652696788311005, 0.9677428007125854, 0.3656364977359772, 0.8064606785774231, -0.10435043275356293, 0.22284463047981262, -1.7390048503875732, -1.0105113983154297, -0.18318112194538116, 1.1356439590454102, -0.28308743238449097, 0.04409345984458923, -1.3164271116256714, -0.12850745022296906, 0.6109879612922668, -1.262120246887207, -2.3336849212646484, 0.8303175568580627, -0.6822680830955505, 0.5789405703544617, 1.504562497138977, -0.8722909092903137, -0.30843037366867065, 0.2507658004760742, 0.6165487766265869, -1.4662173986434937, -1.1164487600326538, -0.6965481042861938, 2.679417371749878, 1.135194182395935, 1.8809531927108765, -0.1725819706916809, 0.0616704523563385, 0.5863562822341919, -0.04082095995545387, -0.3783273696899414, -2.6735992431640625, 1.5408170223236084, 1.8064290285110474, 2.3537986278533936, -0.019477659836411476, -0.5009629726409912, -0.0579642690718174, 0.6311826705932617, 0.6885418891906738, 0.7571291327476501, -1.3465254306793213, 1.0771777629852295, -0.6764874458312988, 0.23378197848796844, 0.7009112238883972, -0.49834737181663513], [-0.840431272983551, -0.8680086135864258, 1.0642459392547607, -1.484607458114624, 0.18652018904685974, 1.3792212009429932, 0.49738210439682007, 0.7312497496604919, -1.4885255098342896, -0.8486846685409546, -0.4036988317966461, -0.939468502998352, -0.9801572561264038, -0.20402580499649048, -0.14626066386699677, -1.1085342168807983, -0.8717178106307983, -0.429586797952652, -0.7128856778144836, -0.3281998932361603, 0.2844655215740204, -1.1078696250915527, 0.6005353331565857, -0.4269088804721832, -0.9862459301948547, -1.3474493026733398, -0.5607593655586243, 0.6306883692741394, -0.45988982915878296, 0.29578012228012085, 0.49000367522239685, -1.3964154720306396, -0.5505382418632507, -0.6346704959869385, -0.15802977979183197, 0.25700056552886963, -1.8550043106079102, -0.250663697719574, 0.8929727077484131, -0.1835108995437622, -0.31559962034225464, -0.4938160479068756, -0.5503634214401245, 0.7976906895637512, -0.7121299505233765, -0.23640403151512146, -0.8425707221031189, 2.013338327407837, -0.08337898552417755, 0.3092286288738251, 0.7425527572631836, 2.4151153564453125, -0.6477194428443909, 0.5377727150917053, -0.5792776942253113, 1.6587032079696655, 1.1375131607055664, 0.19474594295024872, -0.4758067727088928, 0.7354236245155334, -0.17734107375144958, 1.4781739711761475, 0.9624025821685791, 0.6805227994918823], [-0.10584929585456848, 0.29094362258911133, 0.6612942814826965, -0.8846546411514282, -0.664829671382904, -2.0471394062042236, -0.5422510504722595, 0.7951534390449524, -1.2569551467895508, -0.4090227782726288, -0.7423925995826721, -0.29750847816467285, -0.7680566310882568, 1.1694847345352173, -0.6758100390434265, 1.7686443328857422, -0.6812728047370911, 0.12723346054553986, -1.628090262413025, -0.5803781747817993, -0.3594526946544647, 1.087270975112915, -1.0232970714569092, 0.8684726357460022, 0.2997799813747406, 0.08249685168266296, -1.7783805131912231, -0.8173316717147827, 0.00022067893587518483, 1.3981499671936035, 0.6262268424034119, 2.090416431427002, -0.4487219452857971, -0.8481588363647461, -0.21539701521396637, 2.5206310749053955, 0.4384247958660126, -0.8579914569854736, 0.5885022282600403, -1.1121478080749512, 1.464537501335144, -0.0480986163020134, 0.866047739982605, 0.5900355577468872, -0.11152206361293793, -1.3532499074935913, 0.7727037072181702, 0.8221715688705444, 0.8915404081344604, 1.2880667448043823, -1.614241600036621, -0.7108075618743896, 1.2046866416931152, -1.9025049209594727, -0.3698028326034546, -0.8399617075920105, -0.4675999581813812, -0.5099558234214783, 0.400147944688797, 0.4945741891860962, 1.1378940343856812, 2.1559064388275146, 0.2450779229402542, 0.5372046232223511], [-1.1747474670410156, 1.5384492874145508, -0.6807759404182434, -1.1774086952209473, 1.465554118156433, 0.689841091632843, 0.9618081450462341, 0.059979524463415146, -0.09888149052858353, 0.33898648619651794, -0.546571671962738, 0.6735551357269287, 0.9273093342781067, 0.5698027610778809, 1.7628343105316162, -1.4106974601745605, 0.18213820457458496, -1.8731224536895752, -0.27119603753089905, -1.537387490272522, 0.513226330280304, 0.2851870059967041, 1.9337891340255737, -0.747816264629364, -0.6119893193244934, -0.461717426776886, 0.6504140496253967, -0.20694324374198914, -0.4273549020290375, 1.428378701210022, -0.4871485233306885, 0.7165924310684204, -0.8467772603034973, 0.3037828505039215, 1.7564153671264648, 0.4728630781173706, 0.2511237561702728, -2.6054184436798096, 0.6882734298706055, -1.0455610752105713, -1.1815004348754883, -0.21103063225746155, -0.6379278898239136, -0.22962239384651184, 0.3075051009654999, -1.9091650247573853, -1.4039114713668823, -0.8630664348602295, -0.5843279361724854, 0.43338194489479065, 0.1974238008260727, 0.11248801648616791, 0.780339777469635, -0.19643567502498627, -0.8517397046089172, -0.6935804486274719, 0.07136273384094238, -0.45255008339881897, -0.21244896948337555, -1.1466622352600098, -0.44684553146362305, 0.630244255065918, -0.2901577353477478, 1.571763277053833], [-0.37995007634162903, 1.6767427921295166, 0.4005984961986542, -0.35986682772636414, 0.6702718138694763, -1.4933207035064697, -0.18472933769226074, -1.3672683238983154, 0.3513851463794708, 0.3316689431667328, -0.4820803105831146, 0.05506223067641258, 0.250978946685791, -0.2078365981578827, 0.4067574739456177, -1.3234995603561401, -0.09940111637115479, 0.416041761636734, 0.06638672947883606, -0.8948235511779785, 0.10202451795339584, -1.0082945823669434, -0.027445895597338676, 0.5853813886642456, -1.0484380722045898, 1.3823940753936768, -1.075495958328247, 0.18446527421474457, -0.8499634265899658, -0.006532528903335333, 1.2504024505615234, 0.25653135776519775, 0.8978652954101562, 1.1701631546020508, 0.48959073424339294, 0.7295629978179932, -0.1249891147017479, -0.6844797134399414, -0.3198007643222809, -1.4923322200775146, -0.016997091472148895, 1.465375542640686, -0.9709187150001526, -1.0487135648727417, 0.40677976608276367, -1.3034887313842773, -0.27853479981422424, 0.18988560140132904, 0.891794741153717, 0.34231409430503845, -0.5147667527198792, 0.501518964767456, 1.862080454826355, 1.9727048873901367, -1.0969120264053345, -0.9226220846176147, -0.886492908000946, 1.1530177593231201, 0.7642138004302979, 1.6772905588150024, -0.2817276418209076, -0.22726134955883026, 0.23736727237701416, 0.12186658382415771], [-0.18876148760318756, -0.27351030707359314, -0.12957808375358582, -0.8770182728767395, -0.25260627269744873, -0.19131618738174438, 0.2993665635585785, -0.4112979471683502, -1.0248783826828003, -1.419094204902649, -0.6728944778442383, 0.23506046831607819, 0.07298917323350906, 0.1729912906885147, -1.0168235301971436, -1.6849771738052368, -0.9663383960723877, -1.0730191469192505, -0.251997172832489, 1.7988245487213135, 1.1367179155349731, 1.3372095823287964, 0.1458684206008911, -0.5631482005119324, 1.4063833951950073, 0.94612056016922, -0.12699536979198456, 0.7617955207824707, -0.9647495746612549, 0.7732492089271545, 2.1685187816619873, -0.8606317639350891, 1.1237547397613525, -0.7909053564071655, -2.0428671836853027, 0.008413554169237614, -1.1306307315826416, -0.6401444673538208, 1.0700795650482178, -0.7085162401199341, 0.45691362023353577, 0.1975121796131134, -0.7244274020195007, 0.2572164833545685, -1.7370461225509644, 1.5201964378356934, 0.11366234719753265, 0.765419065952301, 2.262998342514038, -1.6299757957458496, 0.5494862794876099, -1.4743863344192505, -1.3138413429260254, 1.7327808141708374, -0.3764194846153259, -1.2322027683258057, -0.3381010591983795, 1.0042660236358643, 0.12694905698299408, 0.4288187325000763, -0.10533323884010315, 0.8810175061225891, 0.7606019377708435, -0.05273682251572609], [0.28925463557243347, 0.6442788243293762, -0.13246634602546692, -3.677563190460205, 1.4103950262069702, 0.9007248878479004, -0.27318763732910156, -0.21194034814834595, -1.3855060338974, -0.2613210082054138, -0.43178591132164, -0.8114085793495178, 0.46279722452163696, 0.40331411361694336, 2.488621473312378, -0.5447076559066772, 0.030369143933057785, 0.47128432989120483, 0.6016769409179688, -0.46947038173675537, 0.6277405619621277, -0.5209912061691284, -0.7625059485435486, 0.04870506003499031, 0.45750316977500916, -0.015337205491960049, -0.3864310383796692, 1.2140415906906128, 0.5317367911338806, -0.34032198786735535, -2.188716411590576, 0.0018364554271101952, -0.1742217093706131, 0.2082459181547165, -1.3975528478622437, -0.8288183212280273, 0.5193747282028198, 0.5026729106903076, 0.03694760799407959, 0.0994434580206871, 0.3369772136211395, -1.2570675611495972, -0.05968857929110527, 0.33264634013175964, -1.6195811033248901, -1.599538803100586, -0.6227872371673584, 1.0386863946914673, 0.7846658229827881, -0.030409835278987885, -0.7761887311935425, -0.20455744862556458, -1.261939525604248, 0.14782457053661346, -0.15832757949829102, -0.5984550714492798, -0.7076273560523987, -0.34283536672592163, -0.5962409377098083, 0.14009450376033783, -1.0296376943588257, 0.017276007682085037, 0.2410239577293396, -0.06376740336418152], [-1.0143595933914185, 1.2046555280685425, 1.2607543468475342, 1.336233139038086, 1.2937716245651245, -0.060664817690849304, -0.19908517599105835, -1.1900182962417603, 1.1405701637268066, -0.9518067240715027, -2.4813051223754883, -0.6974929571151733, -1.1604946851730347, -1.1031122207641602, -0.29740846157073975, 0.4638054370880127, 0.5673673748970032, 0.29853373765945435, 0.008283407427370548, 1.1324570178985596, 0.5502820611000061, 0.63803631067276, -0.7529001235961914, 0.7274490594863892, -1.2227797508239746, 0.2220008671283722, 0.9686664938926697, 0.13679495453834534, -1.3096306324005127, -0.744762659072876, -0.024930093437433243, -1.2016316652297974, -1.3488308191299438, -0.40161848068237305, 1.0892701148986816, -1.5401661396026611, -1.6904903650283813, 0.9982296228408813, 0.007098645903170109, -0.9315157532691956, -0.9221194982528687, -0.6720162630081177, 1.3875465393066406, -0.6326389908790588, 1.041422724723816, 0.3679620623588562, -0.9446089863777161, 2.806178569793701, -1.4078593254089355, -0.493313729763031, -0.946383535861969, 0.47062888741493225, 0.9720211029052734, 0.09760093688964844, 0.5481138229370117, -0.31364887952804565, -1.3035337924957275, 0.4875490069389343, -0.33013302087783813, -0.9463210701942444, 0.22393465042114258, -1.6368687152862549, 0.3420025408267975, 1.2653752565383911], [0.18767106533050537, 0.5714141130447388, -1.0356957912445068, -1.7930439710617065, -0.6535409092903137, 0.4242516756057739, 1.9780060052871704, 0.8342596292495728, -0.20073087513446808, -0.0430128313601017, -0.5331662893295288, 0.25062689185142517, 0.4561667740345001, -0.4968663454055786, -0.6735326647758484, -1.258772373199463, -2.423901319503784, -0.19072650372982025, 0.32541221380233765, 0.1973290741443634, -0.8079307079315186, 0.688447117805481, -0.25936293601989746, -1.1028627157211304, -0.21741031110286713, 1.1894145011901855, 0.38668933510780334, -0.9211823344230652, -0.6556261777877808, -0.6519038677215576, -0.6979627013206482, 0.012307570315897465, -0.3785271942615509, -1.0816996097564697, 1.2931441068649292, -0.8331930041313171, -0.6306882500648499, -1.0460996627807617, -0.125584676861763, 0.11701114475727081, -0.11399272084236145, -0.1626550704240799, -0.902400016784668, -0.32497429847717285, 0.8096924424171448, -0.734122633934021, -1.841548204421997, 0.973742663860321, 1.5090512037277222, 1.4365822076797485, 0.09483443945646286, 0.6634772419929504, 0.2935912609100342, 0.8464187383651733, -0.8876933455467224, -0.1400439590215683, -2.3209497928619385, 0.6227373480796814, 0.8196946382522583, -0.9554240703582764, -1.9967385530471802, -1.779761791229248, 0.955690860748291, 0.9366099238395691], [0.5512099862098694, 1.6278165578842163, -2.303283929824829, -0.9645408987998962, 0.7463545203208923, 1.2597277164459229, 0.6891629695892334, 1.3087056875228882, 2.7404184341430664, -0.43151095509529114, 1.588531255722046, -1.4449139833450317, -0.7116775512695312, -0.5545107126235962, 1.0522282123565674, -1.1562389135360718, -0.8836831450462341, 0.26716551184654236, 0.6506934762001038, -0.9260939955711365, 1.199977159500122, -0.9421672821044922, 1.4780540466308594, -0.43013662099838257, -0.021583644673228264, 0.1200142577290535, -0.6266834735870361, -0.939685046672821, -0.26952439546585083, 1.119053840637207, -0.7190223336219788, 0.8050335049629211, 0.07083945721387863, 0.3645671308040619, 0.674363911151886, -0.11115538328886032, -1.3727513551712036, 0.02815878391265869, 0.9127159118652344, -1.0837316513061523, -0.14562663435935974, 0.5890999436378479, -0.15883465111255646, 1.8257410526275635, 0.46090421080589294, -0.17263594269752502, -0.10417550802230835, -1.6784228086471558, -0.7395403981208801, -2.4595985412597656, 0.4275083839893341, 1.0204309225082397, 0.2681179940700531, -0.5788001418113708, -1.0910700559616089, 0.5189670324325562, 0.4688504934310913, 0.7967767119407654, 0.9893293380737305, -0.07578301429748535, -0.7358267307281494, 0.7914713025093079, 0.3673461377620697, -1.1195361614227295], [1.1477296352386475, -0.255199670791626, 1.805053472518921, 0.4725373387336731, -0.7035239338874817, 0.29248493909835815, 0.3286438286304474, -1.1088272333145142, 1.1360129117965698, 0.4994969964027405, -1.4257159233093262, -0.6017882227897644, -0.4365929663181305, 1.1812140941619873, -1.3976304531097412, -0.3603251874446869, 0.5910151600837708, 0.7674656510353088, 0.09603532403707504, -0.9157090187072754, -0.6153737306594849, -0.3430989682674408, 0.24148224294185638, -1.4775614738464355, -0.24249742925167084, 1.7833442687988281, -0.5971621870994568, 0.3438987731933594, 0.9081498384475708, -0.25235629081726074, 0.05479076877236366, 0.5040023326873779, 0.06797455996274948, 1.535469651222229, 0.4136563837528229, -0.20105323195457458, -0.08196282386779785, -0.5315729379653931, 1.0990153551101685, -0.27340760827064514, -0.778457522392273, -1.733135461807251, -0.6415389180183411, 0.4204515218734741, -1.1306871175765991, 1.9774706363677979, 0.5595810413360596, -0.21060778200626373, 0.44784659147262573, 0.8176882266998291, 0.43128398060798645, 0.8753689527511597, -1.130718469619751, -1.947595238685608, -0.13292892277240753, -0.8829785585403442, -0.9623786211013794, 0.534445583820343, 0.4941822290420532, 1.3648484945297241, -0.19816051423549652, 0.20499171316623688, 0.16673171520233154, -1.0146358013153076], [2.207063913345337, -2.1206846237182617, 1.0761466026306152, -1.8011739253997803, -1.1097114086151123, 0.5664637088775635, 1.3988925218582153, -0.3253680169582367, -1.1170012950897217, -0.19326534867286682, 1.3256111145019531, 0.21482355892658234, -2.4956912994384766, 0.13716161251068115, 0.7865831851959229, -1.1850776672363281, -0.1205194741487503, 2.455857038497925, -0.7056233882904053, 0.23051626980304718, 2.2807960510253906, -1.1205812692642212, 0.7217349410057068, 0.3985411524772644, -0.263307124376297, 0.40232542157173157, -1.0355945825576782, -0.2333720177412033, 0.451170414686203, 1.7423796653747559, -1.0207046270370483, 0.816449761390686, -0.3620281517505646, -1.0900698900222778, 1.3304009437561035, 1.8985003232955933, 0.3271634578704834, 1.125885248184204, -0.1122584417462349, 0.10176758468151093, -1.1304631233215332, -0.09417247772216797, -1.7427096366882324, -1.1334025859832764, -1.0569125413894653, -1.7946815490722656, 0.2842106223106384, -0.4353252053260803, 0.7686445713043213, -0.5069998502731323, 1.4600881338119507, 0.01756298542022705, -1.3879544734954834, -0.19675500690937042, 0.15028493106365204, -0.4210144579410553, -0.9670820236206055, 0.9873484969139099, 1.3237625360488892, 0.3771727681159973, 0.4850406050682068, -2.020247459411621, 0.5963312387466431, -1.2340120077133179], [0.1708502471446991, 0.6767242550849915, 1.281633973121643, -1.574660301208496, -0.3775481879711151, 0.7025207281112671, 0.6281701922416687, -1.4197783470153809, -0.2811097204685211, -1.5780926942825317, 1.6659303903579712, -1.3410992622375488, -0.8269370794296265, -1.2037909030914307, 0.21799694001674652, 0.2429104447364807, -0.2357776015996933, 1.8668755292892456, 0.36931484937667847, 0.038904935121536255, 1.223670482635498, -0.2164398580789566, -1.9560840129852295, -0.005039515905082226, -0.05145925655961037, 0.7723647952079773, 1.7050983905792236, -0.9081461429595947, 0.34375643730163574, 1.0950901508331299, -1.0852899551391602, 1.4618970155715942, 0.7038697600364685, 0.9896281957626343, -1.3606619834899902, -0.26948463916778564, -0.16580885648727417, 2.66692852973938, 0.7204678654670715, -0.4326997697353363, -1.8873257637023926, 0.7200002074241638, -0.21717503666877747, -1.4297057390213013, 2.221391439437866, 0.5055188536643982, 0.24417780339717865, 0.07990487664937973, 1.9113811254501343, -2.003812551498413, -1.197209358215332, 0.6986497640609741, -1.8500055074691772, 0.4583300054073334, -0.3395960032939911, -0.16026988625526428, 1.0634928941726685, -1.3040013313293457, -0.8966208100318909, -1.9645233154296875, -0.7981828451156616, -0.7266421318054199, -1.2011159658432007, -0.5131609439849854], [1.1587594747543335, 1.5044312477111816, 1.5266385078430176, -1.6635756492614746, -1.238587498664856, -0.04893672466278076, -0.16778801381587982, -1.2080718278884888, -0.047730617225170135, 0.0989319309592247, 0.9270932078361511, -1.706662893295288, -0.5437919497489929, -1.1820467710494995, -0.47350943088531494, 0.550098180770874, -1.029691219329834, -1.2198150157928467, -0.28056949377059937, -0.7962711453437805, -0.7525877952575684, 0.4281788468360901, -0.7735434770584106, 0.31674903631210327, 0.4600163400173187, 0.47338590025901794, 0.2956448197364807, -0.7058579325675964, 1.5850865840911865, 0.6452500820159912, 1.5960588455200195, -0.41573119163513184, -0.25148916244506836, 0.8322715759277344, -1.0133002996444702, 0.8139645457267761, -1.5404367446899414, 2.1595163345336914, 0.23391170799732208, -0.9468708038330078, -1.6023246049880981, -0.8939306139945984, -1.367979884147644, -0.20900839567184448, -0.07822371274232864, -1.722354531288147, -0.8883814811706543, 0.7567691802978516, 1.0274842977523804, -1.412493348121643, 0.39703789353370667, 2.4153969287872314, -0.05731819570064545, -0.5270788669586182, -0.18817637860774994, -2.0626163482666016, -0.3926512598991394, -1.9160544872283936, -0.6247755885124207, -0.4171299636363983, 1.301543951034546, -0.5430783629417419, -1.0521907806396484, -0.38346654176712036], [0.8998568654060364, -0.18971410393714905, 0.08892374485731125, -0.5456118583679199, -0.3301607072353363, 0.28987711668014526, -0.3557063341140747, 0.9103811383247375, 0.6792321801185608, 1.8812825679779053, -1.2351303100585938, -0.5090218186378479, -1.367089867591858, -0.25862032175064087, -0.9397315382957458, 0.4989030659198761, 1.3393784761428833, 0.2940193712711334, -0.3920523524284363, 2.0480716228485107, -0.8741798996925354, -1.060903549194336, 0.9781917929649353, -2.1741883754730225, -0.19463451206684113, 0.2563591003417969, 0.6855291128158569, -0.2237549126148224, 0.529566764831543, -0.15079069137573242, 2.357300043106079, 0.7818594574928284, -0.3192742168903351, -0.7295788526535034, 0.7941409945487976, 1.5169508457183838, -0.1790395975112915, 0.23459281027317047, -1.3500107526779175, 0.10867926478385925, -0.05661576986312866, 2.6639063358306885, 1.9874340295791626, 0.02553693950176239, -0.7842921018600464, 0.27010467648506165, 0.5517828464508057, -0.192130446434021, 0.6075619459152222, -0.06095618009567261, 0.8267064690589905, -1.139737844467163, -0.15049099922180176, -3.3478922843933105, 1.2320783138275146, 0.021615777164697647, -1.163698673248291, -0.03783893212676048, 0.25673702359199524, 0.2828232944011688, -0.3782655894756317, 0.14300589263439178, -0.22831638157367706, 0.338410884141922], [-1.0386415719985962, 0.13054311275482178, -0.6091641187667847, -0.7756577730178833, 0.26139646768569946, -0.2556546628475189, -2.355905771255493, -0.5668267011642456, 1.4535706043243408, -2.393364429473877, 0.4648669958114624, 0.009477525018155575, 0.3045916259288788, 1.4438894987106323, -0.48478779196739197, -0.6586883068084717, -1.8882122039794922, 1.0971652269363403, -1.2266703844070435, -0.46321460604667664, -0.3854968845844269, 0.947482168674469, 0.2764562666416168, 0.7423059940338135, -0.06922397017478943, 0.11301348358392715, 0.4721044600009918, -0.32638007402420044, 1.4784413576126099, -0.575065016746521, 2.7284891605377197, 0.2722204327583313, -0.01939757540822029, 0.7534162402153015, -0.4018382430076599, 0.9463603496551514, 0.6862537860870361, -0.4132969081401825, 0.2997719943523407, 0.7510051727294922, 0.9728617668151855, 0.8458693623542786, -0.2617901861667633, 0.1617947816848755, 1.1255717277526855, -1.7857393026351929, 0.25225263833999634, -1.9276487827301025, 0.3269224762916565, 1.6658670902252197, -0.8144317865371704, 0.8805490136146545, 0.31843888759613037, -0.25207293033599854, 0.2004852592945099, -0.3712459206581116, 0.13837750256061554, -1.9043618440628052, -0.2830985188484192, -0.5469414591789246, 0.022907886654138565, -0.6574166417121887, -0.1703455001115799, 0.4787379801273346], [1.106683611869812, 1.3735212087631226, -1.4195631742477417, -1.4022886753082275, -0.14715994894504547, 0.5911953449249268, 1.3832383155822754, -0.2975466549396515, 1.571901798248291, -0.5347551703453064, 1.496814489364624, 0.5320600867271423, 1.2122408151626587, 0.2648618519306183, -0.49790650606155396, -0.1646830141544342, 0.8758256435394287, 0.5122465491294861, -1.490187406539917, 0.3980884850025177, 0.44405531883239746, -1.3487557172775269, 1.1661242246627808, -1.4756970405578613, 1.058538556098938, -0.5628582239151001, -0.7451543211936951, -0.49915820360183716, 0.6135203242301941, -1.2456382513046265, 0.40051183104515076, 1.2859351634979248, -0.7137962579727173, 1.0100071430206299, 0.3647918105125427, 0.26620903611183167, -0.1728135347366333, -1.6404355764389038, 1.2300939559936523, -0.449950635433197, 1.9999094009399414, 0.3421761095523834, -0.07413874566555023, 0.1373547911643982, 0.7614569067955017, 0.2700246572494507, -0.5518478751182556, -1.3920530080795288, 1.7797610759735107, -0.04058234021067619, -0.8706398010253906, 0.433763712644577, 0.30447864532470703, -0.58280348777771, 0.022099042311310768, -1.5383774042129517, 1.0416945219039917, 1.2404088973999023, -0.6668744087219238, -1.0151820182800293, 0.5388014912605286, -1.0429341793060303, 0.4338935613632202, 1.4426794052124023], [0.10074576735496521, -0.3404475152492523, -0.09929554909467697, -0.8092777729034424, -2.231114625930786, -0.015047719702124596, 0.11259406059980392, 1.7898821830749512, -0.4762357175350189, -0.545270562171936, 0.5143234133720398, -1.0058852434158325, 2.668654203414917, 0.5549828410148621, 1.0855448246002197, -0.5566322803497314, 0.6562246084213257, -0.054066579788923264, -0.6896103620529175, -0.1599707454442978, -0.3202735483646393, -0.467367947101593, -2.2285141944885254, 0.16063661873340607, -0.2151140421628952, 1.9857308864593506, -0.06542381644248962, -0.49492010474205017, 1.0261383056640625, 0.7229735255241394, 0.6074135899543762, -0.11362394690513611, 0.0471971333026886, -0.9997843503952026, 0.8387317061424255, -0.5476570129394531, -0.6819384694099426, 0.02958814613521099, 0.9684426784515381, 0.7954630255699158, 1.7685660123825073, 0.34916582703590393, -1.4742469787597656, -2.4849956035614014, 1.5360206365585327, -1.0718828439712524, 0.3451758623123169, 0.6809625029563904, 0.5086383819580078, 0.0569128543138504, -0.2644244432449341, 3.1070568561553955, 0.26992908120155334, 1.2830380201339722, 1.108896017074585, -0.31972095370292664, 0.797730028629303, 0.5343338847160339, 0.4478672742843628, 1.062006950378418, 0.37291234731674194, -1.4695708751678467, 0.12148916721343994, -0.4541289508342743], [1.3804643154144287, -0.8459774255752563, 0.2126161754131317, -2.0611588954925537, -0.04637005180120468, 0.6830647587776184, -2.6394739151000977, 0.3960547149181366, -0.6670092940330505, 1.1993438005447388, 0.10565586388111115, -0.3677442967891693, -0.01979241892695427, -1.613204836845398, -0.8767085671424866, 0.546385645866394, -0.5534337162971497, -0.16190370917320251, 0.5204249024391174, 0.1267072558403015, -1.394020915031433, 0.6964849233627319, -0.8085974454879761, -1.4551537036895752, 0.5969697833061218, -1.496504783630371, 1.2862533330917358, 0.3074645698070526, -0.613049328327179, 1.268458366394043, -2.3040261268615723, 0.5104977488517761, 0.6163588166236877, 0.4925396144390106, 0.1433446705341339, -0.8677828907966614, 0.06804955005645752, -1.2447742223739624, 1.0789992809295654, 0.7738105654716492, 0.5024813413619995, 0.038563117384910583, 0.5281941890716553, 1.9545092582702637, -1.0965229272842407, -0.7371028065681458, -1.3718422651290894, -0.6050721406936646, -1.6155579090118408, -1.334241509437561, -1.3996078968048096, -0.338634729385376, 0.625423789024353, -1.2358959913253784, 0.11701032519340515, 1.5074557065963745, 2.768742084503174, -0.8530223369598389, 0.6750061511993408, -1.187837839126587, 0.1926349550485611, -2.786020517349243, 1.065909743309021, -2.984145402908325], [1.243949294090271, -1.529151201248169, 0.15395842492580414, -0.5623965859413147, 0.5076669454574585, 0.5259994268417358, -0.5087239146232605, 0.47414469718933105, -0.26535937190055847, -0.5858338475227356, -0.8300574421882629, -2.388429880142212, 0.013837653212249279, -0.7662197947502136, 1.8241381645202637, -0.20504538714885712, 0.4367334544658661, 0.9944205284118652, -0.328905314207077, 1.1885979175567627, 1.7124853134155273, -0.28915613889694214, -1.796194314956665, -0.12941589951515198, 1.00325608253479, 0.534511387348175, -0.28977978229522705, 2.5686426162719727, -0.22297058999538422, -0.14906984567642212, 0.7523977756500244, -0.8865602612495422, -1.476639747619629, -1.1521755456924438, 1.0060803890228271, 0.4989686906337738, -0.1794796586036682, 2.3584742546081543, 0.20424509048461914, -1.7500784397125244, -0.18873891234397888, -1.536863923072815, -0.03552291914820671, -1.2185630798339844, -0.0765591412782669, 0.2946661114692688, -0.780279815196991, 2.304062843322754, -0.16629187762737274, -0.7926839590072632, 0.8470173478126526, 1.3340001106262207, -2.2604851722717285, 2.386439323425293, 0.2533904016017914, 0.6235042214393616, 0.869851291179657, -0.6398800611495972, 0.47158312797546387, 1.435518503189087, 0.4997451603412628, -1.1409810781478882, -0.9578514099121094, 1.817879557609558], [0.047960542142391205, -0.01763245463371277, -0.14096567034721375, 0.6354217529296875, -0.043156806379556656, -0.11921151727437973, -0.14548169076442719, 0.09813655912876129, -0.1930898129940033, -0.38380205631256104, 0.1760859191417694, 0.3166787624359131, 0.585673451423645, 1.662412405014038, -1.2288445234298706, -0.2815621495246887, 1.2452762126922607, -1.3725323677062988, 0.14197483658790588, 0.5719250440597534, 1.3386317491531372, 1.647481083869934, -0.9706394076347351, -0.6168213486671448, -0.007504043634980917, 0.32440319657325745, 1.0572837591171265, -1.7129714488983154, 1.6628601551055908, -0.36738502979278564, -1.0080727338790894, -0.020384611561894417, -0.570011556148529, 0.6550081968307495, 2.245577335357666, 0.4024435877799988, -0.5523410439491272, -3.1586968898773193, -0.7011206150054932, -1.1339952945709229, 0.7402237057685852, -0.029757333919405937, -1.3155097961425781, -1.5531476736068726, -0.8923187851905823, -0.09974014759063721, 0.4482031464576721, -1.0405675172805786, -0.933617115020752, 0.6717656850814819, -1.362914800643921, -0.48208898305892944, -0.2949746251106262, -1.635535478591919, -1.03880774974823, -0.07116395980119705, -0.20403780043125153, -0.10620653629302979, -0.37645384669303894, -0.5122885704040527, -0.18743568658828735, -1.6907763481140137, -2.550016164779663, 0.8194507956504822], [1.1113243103027344, 1.251578450202942, -0.8447244763374329, -0.3917672038078308, 0.3605187237262726, -0.18892180919647217, -0.9203833341598511, -0.7725787162780762, 2.473567485809326, 0.2343386560678482, 1.593512773513794, -0.28151971101760864, 0.9367666840553284, -0.8091852068901062, -1.2508593797683716, -0.06433118879795074, 1.1478450298309326, 0.8840131163597107, 0.8573780059814453, 0.09417546540498734, -0.935172975063324, -2.1349546909332275, 1.6155990362167358, 1.712263584136963, 0.25170955061912537, -0.3837451636791229, 0.36972448229789734, -0.9814724922180176, 0.4588572382926941, -0.8346946835517883, -1.5391405820846558, -1.0908151865005493, 0.11225394159555435, 2.1849887371063232, 0.8257303833961487, -0.15375986695289612, -0.4724094271659851, 0.036997873336076736, 0.9914910793304443, 0.8867424726486206, -1.4991481304168701, 0.526378333568573, 0.673701822757721, 0.3198549747467041, -1.1854119300842285, 2.1878631114959717, -0.08056056499481201, -1.3004093170166016, -0.9438838958740234, 0.03724770247936249, 0.3056793510913849, -0.44451573491096497, 0.4023445248603821, 1.0353062152862549, 1.3962546586990356, 0.20451487600803375, -0.30974456667900085, -0.15882508456707, -0.1163724884390831, 0.7613683938980103, 0.2598686218261719, 1.3708618879318237, -0.2813439965248108, -0.04766678437590599], [-1.3683022260665894, -0.7071853280067444, -0.20144933462142944, -0.20149506628513336, 0.037668000906705856, -0.9531021118164062, -0.8800875544548035, -0.11717891693115234, -0.9442075490951538, 0.12636816501617432, 0.5706605315208435, 0.2749343812465668, 0.9831123948097229, -0.09397222846746445, -0.18387606739997864, 1.3765904903411865, 0.6557022929191589, -1.0478618144989014, 1.4178167581558228, 3.6718380451202393, 0.7779788970947266, 0.5271670818328857, 0.7222335934638977, 0.4682095944881439, -1.2844672203063965, -0.2246031016111374, -2.2049503326416016, 0.23042050004005432, 0.7937942743301392, -0.3979247808456421, -0.19858697056770325, -1.2187141180038452, 0.1595439612865448, 0.5025134086608887, 1.4144887924194336, 0.8031054735183716, -0.19445757567882538, -0.1454956829547882, -0.840843915939331, 1.3950914144515991, -1.0634639263153076, -0.5882208943367004, 0.7737149596214294, 0.46385183930397034, 1.1600074768066406, -1.233141303062439, 0.5181810259819031, -0.8925470113754272, 0.008842810988426208, 0.26840469241142273, -0.5178114175796509, 0.8381190896034241, 0.6764149069786072, -0.2074553668498993, 0.33614206314086914, -0.4489871561527252, -1.5225272178649902, -0.36609771847724915, 0.033936772495508194, -1.5013333559036255, 0.03842100501060486, 0.1112586185336113, -0.3628009557723999, 0.17520758509635925], [2.256859540939331, -0.8923429846763611, 0.04165852814912796, 1.1957283020019531, -0.39092594385147095, -0.7789283990859985, 0.45367079973220825, 0.7525646090507507, 0.5211967825889587, 0.8552982807159424, 1.221419095993042, 0.546900749206543, 0.2229549139738083, 0.8937531113624573, 0.35803255438804626, 1.170208215713501, -0.45935767889022827, 0.06564930826425552, -0.8437984585762024, -0.336874783039093, -1.3234822750091553, -0.32913726568222046, -0.11522937566041946, -0.8126310706138611, 0.8136938810348511, 0.4131213426589966, 0.4206714928150177, -1.5240402221679688, 0.6861156225204468, -0.9659309387207031, -1.3020652532577515, -2.781268358230591, -0.9247322678565979, 1.338304042816162, -0.49737414717674255, 0.7297235727310181, -0.25259125232696533, 0.1608080416917801, -1.4515281915664673, -0.0004509049467742443, -1.053231120109558, 1.0495120286941528, -0.14251646399497986, -0.267695814371109, 1.3145794868469238, 1.0747748613357544, -0.14138656854629517, 1.220278263092041, 1.208829641342163, -1.5242722034454346, -0.29742878675460815, -1.3385117053985596, -0.31187090277671814, 0.4697130024433136, 1.7946445941925049, 0.1258653849363327, 0.9020248651504517, -0.08668960630893707, -0.06754626333713531, -0.4314391613006592, -0.7183465957641602, 0.890519380569458, -0.6278026103973389, -0.3706659972667694], [-0.9196407198905945, -0.043663062155246735, -0.5742261409759521, 1.113147258758545, 0.6590937376022339, 0.1335076093673706, -0.23577752709388733, 0.2045576423406601, -0.36438339948654175, 1.2253212928771973, -0.6686272025108337, -0.3205820620059967, 0.8959721922874451, -0.10682740807533264, 0.28177618980407715, -0.7536348104476929, 0.23144936561584473, 0.41805940866470337, 1.646575927734375, 0.3453809320926666, -0.17881827056407928, 1.3999799489974976, -0.5930973291397095, -0.8639487028121948, 0.655961811542511, 3.513892650604248, 0.5718204975128174, 0.6665928959846497, -0.40092816948890686, -0.42002829909324646, -1.6924536228179932, -0.011751900427043438, -0.896831750869751, -0.5096626877784729, -0.3439640998840332, 0.530666172504425, 0.4984952509403229, -1.2403920888900757, 0.7133928537368774, 1.5815997123718262, 2.5076146125793457, 0.1534969061613083, -0.05535615608096123, 0.3893561065196991, 0.01812778227031231, -1.609505534172058, 0.051182202994823456, -1.2649435997009277, -1.1536551713943481, 0.2210998386144638, 0.15984980762004852, -0.9027486443519592, -0.7963923811912537, 2.068758010864258, 0.7567387223243713, -0.40017110109329224, 0.23929572105407715, 0.5291498303413391, -0.9677926301956177, 0.004686644766479731, -0.5027792453765869, 0.2019304782152176, -0.037996526807546616, -0.10507672280073166], [0.6293109655380249, 0.3305147886276245, 0.09028559923171997, 1.142248272895813, -0.20132111012935638, 0.8897222876548767, 1.3468881845474243, 0.5139225125312805, 1.8660244941711426, -0.32018399238586426, -2.001782178878784, -0.030327705666422844, -0.25846797227859497, -0.036627255380153656, -2.5577733516693115, -0.7582439184188843, 0.7778488993644714, -2.16705060005188, 0.8725423812866211, 0.02792992815375328, -0.5110995769500732, -0.479091078042984, -1.5983978509902954, -0.576022207736969, 0.3272190988063812, -1.5797998905181885, 0.5533844232559204, -0.9441991448402405, -0.37759527564048767, 0.20611341297626495, 0.07432196289300919, 1.1219693422317505, -0.3312685191631317, 0.32851722836494446, -1.7502247095108032, -1.0476109981536865, 0.67476886510849, -0.7595507502555847, 0.4322825074195862, -0.7989512085914612, 0.49976903200149536, 0.19905877113342285, -0.2582220733165741, 2.066340684890747, 0.8337224721908569, 0.037405308336019516, -0.5082211494445801, -0.0909532830119133, 0.08477406948804855, -0.3504716455936432, -1.1282548904418945, 1.0802513360977173, 0.17715269327163696, -0.872331976890564, 0.5172401070594788, -0.578274667263031, 1.0365155935287476, 0.9676693677902222, -0.5150376558303833, 0.5940330028533936, -0.8056820631027222, 1.01304030418396, -0.9713600277900696, 1.0898566246032715], [1.9270187616348267, -0.5384035706520081, 0.8860470056533813, -1.2810354232788086, -0.5131595730781555, 0.6566479802131653, 0.025594426319003105, 0.8356030583381653, 0.6529393196105957, 0.7666961550712585, 0.1795593500137329, 0.6180073022842407, -1.1837087869644165, -1.9858052730560303, -0.7305971384048462, -0.9202670454978943, -0.9282249212265015, 0.17758029699325562, -0.12329737097024918, 0.1739472597837448, 1.4220880270004272, 0.8030475974082947, -0.020951688289642334, -0.10318276286125183, -1.1398597955703735, -1.125876545906067, 0.07218965888023376, 0.5319377779960632, -1.3930917978286743, -2.0455307960510254, 1.05320405960083, -1.951843023300171, 1.0611242055892944, -0.670672595500946, -0.14373880624771118, -0.2761862874031067, -2.0992374420166016, 1.1509329080581665, 0.7292661666870117, -0.2842201292514801, -0.18458899855613708, -0.5702780485153198, 0.4587281048297882, 0.9398365616798401, 1.7807776927947998, 0.3153669238090515, -1.1248055696487427, -1.0529764890670776, 0.08150757104158401, 0.7732903361320496, -0.42416664958000183, 1.550266981124878, -1.041581153869629, -0.3413630425930023, 0.3891928493976593, 0.9982264637947083, -1.5614949464797974, 0.6314373016357422, -0.25012123584747314, -1.6252692937850952, 1.2732045650482178, -1.6372839212417603, 0.9530659317970276, -0.17308981716632843], [-0.07551765441894531, -0.45109403133392334, 0.548151433467865, 0.6605578660964966, 0.7553790807723999, 0.44683748483657837, -0.02906777337193489, 1.3543291091918945, -1.3530218601226807, -0.22961606085300446, 1.1252093315124512, -0.7220211029052734, 0.8743206262588501, -0.8596437573432922, -1.2690324783325195, 0.9125470519065857, -1.3507784605026245, -1.445774793624878, -1.5433378219604492, 2.07070255279541, -1.3075331449508667, 0.2309703230857849, -0.0032084255944937468, -0.7731048464775085, 1.2506177425384521, 0.15252713859081268, 0.15127795934677124, 0.5986685752868652, -1.5766853094100952, -1.0758384466171265, 0.020874060690402985, -1.4719114303588867, -1.995182991027832, -0.25566112995147705, -0.6840536594390869, 1.3707208633422852, -0.011666773818433285, 1.1635500192642212, 1.9441769123077393, 0.6643001437187195, -1.3898495435714722, 0.1468368023633957, 0.11653659492731094, -0.6035780906677246, 0.28034472465515137, -0.5392877459526062, 1.4227008819580078, -1.1870149374008179, -0.9652456045150757, 1.406866431236267, 0.03395887091755867, 0.5445048213005066, -1.7245442867279053, -0.32050085067749023, -0.6178129315376282, 1.571153163909912, 0.8894248008728027, 0.6151729822158813, -0.6354212760925293, -1.7162476778030396, -0.35415294766426086, 0.3524834215641022, 1.2928107976913452, 1.419084072113037], [0.5741703510284424, -0.41299760341644287, 1.3414373397827148, 1.7700121402740479, 0.7402855157852173, 0.03271983563899994, -0.6119834184646606, 2.5550248622894287, -0.9923437833786011, -0.008224516175687313, -0.4874008595943451, -0.5139156579971313, -0.9646485447883606, -0.8278370499610901, 0.8013013005256653, 0.4299188256263733, 0.428438276052475, -1.0716819763183594, 0.6222382187843323, -0.46330130100250244, 0.6570518612861633, -0.9506875276565552, -0.35829317569732666, -0.3920447528362274, 0.15664257109165192, -0.13139724731445312, -1.4486838579177856, -0.912649929523468, -0.8039472103118896, 0.1898428350687027, 1.3040366172790527, -0.9138256311416626, -2.5296192169189453, -1.8322118520736694, -1.9073618650436401, 0.22445671260356903, -0.7690879106521606, -0.47470590472221375, -1.7268130779266357, 0.3404582142829895, 1.4251537322998047, 0.3797876238822937, -1.020461082458496, -1.442747712135315, -0.4623447358608246, 0.35553014278411865, 0.6069516539573669, 0.227195143699646, -0.17875131964683533, -0.24294352531433105, 0.911175012588501, -0.12233984470367432, -0.3109278976917267, -2.6373863220214844, 1.6274508237838745, 1.1952965259552002, -1.2134952545166016, 1.9057186841964722, 1.0820435285568237, -1.0056968927383423, -0.5637125968933105, 0.6076358556747437, -1.1524815559387207, -0.587104320526123], [-0.7492756247520447, 2.7050318717956543, -1.178756833076477, -2.9509918689727783, 0.723231852054596, 2.938734769821167, 0.6034688949584961, -0.36735716462135315, 0.8932967782020569, -0.5828462243080139, 1.3872604370117188, 0.3455600440502167, -0.29586389660835266, -1.2709342241287231, -0.2594144940376282, 0.5639200806617737, 0.893523097038269, 1.0242576599121094, 0.9193615913391113, 0.21995720267295837, 0.10602245479822159, 1.7134257555007935, 1.542828917503357, 0.06364312022924423, 0.01948157511651516, -1.2982805967330933, 0.11537441611289978, 0.8667462468147278, 0.31679728627204895, 0.6225236654281616, 2.026198387145996, 0.5364521145820618, -1.3592735528945923, 0.27580544352531433, 0.9249081611633301, -1.459605097770691, -1.6316125392913818, -2.115327835083008, -1.9254285097122192, -0.4676951766014099, 0.4338838458061218, 1.2449339628219604, -1.832301139831543, 0.5018581748008728, -0.19794805347919464, -1.2882660627365112, 0.7160165309906006, -0.7017840147018433, 1.176810383796692, -0.6064680218696594, -1.6842021942138672, -2.626505136489868, -0.05790285766124725, -0.5660041570663452, -0.6376650929450989, 0.40856438875198364, -1.6270169019699097, 1.2571419477462769, 0.5850396156311035, -0.0307354386895895, -0.7210637927055359, -1.8928498029708862, 1.7531439065933228, 2.2764780521392822], [-0.7939193248748779, 0.2856714725494385, 1.5597527027130127, -0.2845098376274109, 0.715929388999939, 0.9838669896125793, -1.4264686107635498, -1.0395647287368774, -0.06460189074277878, -0.24225620925426483, -0.2728869915008545, -0.6713840365409851, -2.0581743717193604, -0.23956608772277832, -0.26880303025245667, -0.04110267758369446, 0.24147477746009827, 0.40434595942497253, -0.9770474433898926, 0.4786962866783142, -1.6902881860733032, 0.684190571308136, 1.304502248764038, 0.43658432364463806, -0.052645493298769, -0.8629372119903564, 0.8707721829414368, 0.2709338963031769, 0.18139812350273132, -0.4008018374443054, 1.0236599445343018, -0.8105857372283936, 0.1933407187461853, 2.708916664123535, 1.1374855041503906, -0.11208709329366684, -0.48444804549217224, 0.8740298748016357, -0.5774083137512207, 0.5691731572151184, 0.08165313303470612, -2.441986560821533, 0.08041956275701523, 1.276809811592102, -1.1670420169830322, 0.02249244414269924, -0.9268337488174438, -0.4441918730735779, 0.7348093390464783, 0.9003258347511292, 1.783087968826294, 0.8794670701026917, -0.39329007267951965, 0.9533928632736206, -0.6087609529495239, -1.4993274211883545, 0.4030749201774597, -0.1682160645723343, -1.2117139101028442, -1.0715196132659912, -0.22512903809547424, -0.040930990129709244, -0.1731124222278595, 2.4100637435913086], [-1.5636179447174072, -1.4209554195404053, -0.27195167541503906, -0.3127025067806244, -0.5017136931419373, 1.1163389682769775, 0.4773992896080017, 0.11057472974061966, 0.4032301902770996, 0.5240820646286011, -0.10527237504720688, 1.035710096359253, 0.8826900124549866, -0.6107328534126282, -0.009049602784216404, -1.1558668613433838, -1.3138511180877686, -0.9894888997077942, 0.18241249024868011, 0.41258567571640015, 0.04656899720430374, 1.8991104364395142, 1.8951293230056763, 1.3859481811523438, -1.3202159404754639, -1.0428131818771362, -0.3421368896961212, -0.3271642327308655, 1.9472397565841675, 0.5189677476882935, 0.540550172328949, 1.4450360536575317, 0.969003438949585, -1.3755921125411987, -1.3665679693222046, -0.8641877770423889, -2.531191110610962, -0.46391668915748596, -0.5210900902748108, 0.08158589899539948, -2.732347249984741, -0.1763431429862976, -0.8490576148033142, -0.15964072942733765, 0.4778100252151489, -0.9851906299591064, 0.8885730504989624, -2.2233469486236572, 0.08096075057983398, -2.033364772796631, -2.2427966594696045, 0.9285953640937805, -1.2023305892944336, -0.03101317770779133, -0.19978199899196625, 1.7247114181518555, -1.6917123794555664, -2.0110859870910645, 0.9681531190872192, 0.6583332419395447, -0.06716910749673843, -0.16633132100105286, -1.6735458374023438, -0.00571201229467988], [0.5875762104988098, 0.07141131907701492, 0.12575098872184753, 1.5474858283996582, 1.7661577463150024, -1.1103812456130981, -0.5621172785758972, 0.17787960171699524, 0.007933864369988441, 0.7202505469322205, 1.4598710536956787, -0.2588076591491699, 0.38282638788223267, 1.0145440101623535, 0.010890166275203228, 1.126922369003296, -1.2172966003417969, -0.9669374823570251, 0.4111853539943695, 0.20780760049819946, 2.1464221477508545, 1.0446454286575317, -0.09223548322916031, -0.015617758966982365, -0.7025915384292603, 0.27572453022003174, -0.4571322202682495, 1.087829828262329, 0.22663363814353943, -1.3305613994598389, 0.830340564250946, -0.00036812483449466527, 0.9779689908027649, 0.5739837884902954, -0.8525828719139099, 1.5276836156845093, -1.5009996891021729, 3.4857635498046875, 1.9244517087936401, -1.6525802612304688, 1.6110748052597046, 1.2380815744400024, -0.1036311611533165, 0.1377173364162445, 1.0528477430343628, 0.8218221664428711, 1.5007562637329102, 0.40075457096099854, -1.2497355937957764, 1.0123066902160645, 1.0420702695846558, -1.6111693382263184, -0.5606454014778137, 0.8318256735801697, -0.8771381378173828, -0.3656579852104187, 0.8825921416282654, 0.031188007444143295, -0.10293811559677124, -1.7345112562179565, -0.8985412120819092, 0.3794859051704407, -2.3415639400482178, -2.2656538486480713], [-1.2908011674880981, 0.9745509624481201, 3.036586046218872, 0.23180390894412994, 1.4777644872665405, -0.5226538181304932, -0.7517329454421997, -0.1522071659564972, -2.0278024673461914, -0.41207268834114075, 0.8789606094360352, -1.034042477607727, 0.7609453201293945, 0.7217504382133484, -0.8747655749320984, -0.5723814368247986, 0.41750526428222656, -0.2148251235485077, 1.2713563442230225, -0.12127316743135452, 0.06541234254837036, 0.2822003662586212, 0.2953912317752838, -1.5514461994171143, 0.2392849326133728, -0.12430371344089508, 0.7041041851043701, 2.37050199508667, 0.14021921157836914, 1.0122199058532715, 0.22511297464370728, 0.7725228071212769, 0.3196473717689514, 0.4437209665775299, 0.2954660952091217, 1.0250848531723022, 1.57371985912323, -0.05350646376609802, -1.0892438888549805, 1.1892240047454834, 0.48356062173843384, -0.48233363032341003, 0.061748865991830826, 0.6466485857963562, -0.007574206218123436, 0.7969631552696228, -2.1108338832855225, 0.22139126062393188, 0.10268385708332062, -0.6279712319374084, 0.12049827724695206, 0.06368333101272583, -0.20600219070911407, -0.06710974127054214, 1.0692591667175293, -0.2591606080532074, 1.6386768817901611, -0.23099927604198456, 1.283747911453247, -1.136604905128479, 0.7960549592971802, 0.35694894194602966, -1.076189637184143, 0.575240433216095], [1.144243597984314, 0.9573732018470764, 0.6122112274169922, 1.1681710481643677, -0.24713042378425598, 0.5321037173271179, 0.6277973651885986, -0.16540081799030304, 0.5801712274551392, -0.13265098631381989, -0.053738031536340714, 0.44809046387672424, -0.5411834716796875, -0.08043971657752991, 1.4053906202316284, 0.05568643659353256, 0.15734440088272095, -0.3112656772136688, 0.23784732818603516, 0.9642106294631958, 0.021505206823349, 3.221550464630127, 1.4480819702148438, 1.555516004562378, -1.481635570526123, -0.2629060745239258, -1.7813372611999512, 1.1024724245071411, -0.31436723470687866, -1.6635348796844482, 0.12465861439704895, -1.3410073518753052, 1.8985674381256104, 0.21862584352493286, 0.880204439163208, -1.1862250566482544, 0.8611871004104614, 0.11894819140434265, -0.35355839133262634, -0.19307340681552887, 0.693181574344635, -1.5932433605194092, 0.23724070191383362, 0.3554738759994507, 0.2924014627933502, -0.5425201654434204, 0.785516619682312, 0.4044683873653412, 0.3922254145145416, 1.2792136669158936, 0.6834859848022461, -0.32939717173576355, -1.2157001495361328, -0.36281001567840576, -0.13294318318367004, -0.003512713825330138, -0.8925778865814209, -1.200173020362854, -1.1608974933624268, 1.7096654176712036, -2.235595226287842, -0.09546715021133423, 0.3397071063518524, -0.741974949836731], [0.8868438601493835, -1.7201448678970337, -0.9831551909446716, -0.44501161575317383, -1.6076267957687378, 0.531502366065979, 0.1748049259185791, -2.195573329925537, -0.13993780314922333, 0.301315039396286, -0.14964154362678528, -0.5260637402534485, -1.4641797542572021, -0.834848940372467, 0.48716869950294495, -0.3550930917263031, 0.2217961847782135, 1.3582435846328735, -0.44178855419158936, -2.0255794525146484, 1.1113348007202148, -0.34486743807792664, 1.2353981733322144, -1.0668156147003174, 1.7411830425262451, -0.9128630757331848, -0.8360223770141602, 0.4078907072544098, 0.23887042701244354, -0.3660396635532379, -0.38747790455818176, -0.3461591303348541, 0.36697524785995483, 1.7509207725524902, 1.1433683633804321, 0.3672817647457123, -1.367539882659912, -0.791113555431366, -1.5797332525253296, -1.3998559713363647, -1.064834475517273, 1.5704704523086548, 0.4940739870071411, 0.1327783316373825, 0.2295616716146469, -0.5563881993293762, -2.3873019218444824, -1.1362179517745972, -0.9380528926849365, 0.6000755429267883, 0.36119288206100464, -0.9784554839134216, 1.234304428100586, -0.09404931217432022, -1.2657805681228638, -0.48588740825653076, 0.6040725111961365, 2.134059190750122, 0.2170427441596985, -1.1188281774520874, 0.24904455244541168, 0.8391816020011902, -0.9958264827728271, -1.5811213254928589], [-0.7728660702705383, 0.08216255158185959, -0.527390718460083, 1.2392433881759644, -0.11663150787353516, 2.2352731227874756, 1.1128896474838257, 0.5338939428329468, 1.2402825355529785, 1.2461549043655396, 0.02594299428164959, -1.063153862953186, 0.4731365144252777, -0.16011400520801544, 0.6185668706893921, 0.7664844989776611, -0.9089358448982239, 1.4276611804962158, -0.2838030159473419, 0.8799200057983398, 0.6404320001602173, -1.7433831691741943, -1.9099735021591187, -0.10247807949781418, -0.5405226945877075, 1.8521982431411743, -0.8590592741966248, -0.3511354625225067, -0.781997561454773, 0.6056879162788391, -0.7022064328193665, -1.4814542531967163, 1.6409140825271606, -1.5694118738174438, -0.11688730865716934, 0.8838293552398682, -1.1896673440933228, -0.28299546241760254, -0.9676617980003357, -0.2723126709461212, 0.647720456123352, -0.506614089012146, 1.4974249601364136, -0.7221238613128662, -0.34671133756637573, 0.04789469391107559, -0.11910814791917801, -1.6757508516311646, -0.9184486865997314, 0.11942722648382187, -0.9887660145759583, 1.2060823440551758, 1.2357347011566162, 0.8030092120170593, -1.3049949407577515, -1.2359713315963745, -0.2827324867248535, -0.4188250005245209, 0.013077525421977043, -0.597304105758667, -1.3233301639556885, -0.3403182327747345, -1.443264126777649, -2.184657335281372], [1.0245312452316284, 0.2516501843929291, 0.64962238073349, -0.04471768066287041, 0.6313560009002686, -0.40406620502471924, 1.0299469232559204, -1.3863402605056763, 0.2684119641780853, 1.3081191778182983, 0.2765885293483734, -0.036268964409828186, -0.4418237507343292, 1.9808987379074097, 0.18545986711978912, 1.7341800928115845, -1.735183835029602, 0.10582854598760605, 1.2097851037979126, -0.5838711857795715, -2.3328964710235596, 1.07744562625885, -1.6309486627578735, 1.0607619285583496, 1.1869606971740723, -0.9655519127845764, 0.3389262855052948, -0.5722758769989014, 1.9859938621520996, 0.795609176158905, -0.8488976359367371, 1.0376815795898438, -0.8849459290504456, 0.7363804578781128, 1.3165351152420044, -1.595947504043579, 1.2336440086364746, -0.13562463223934174, -0.28358742594718933, -1.9220874309539795, 0.1574658751487732, 1.1697343587875366, -0.3835161030292511, -0.8890538215637207, 0.04674622789025307, 0.625424861907959, -0.016584914177656174, 0.019263358786702156, 2.2778854370117188, -0.8639788031578064, -0.6684893369674683, 0.4033244252204895, -2.7760987281799316, -1.703150987625122, -1.5735079050064087, -2.272027015686035, -0.8931543231010437, 0.36167243123054504, 0.5356431603431702, -0.02867553010582924, 1.5950117111206055, -1.0988534688949585, 0.15085195004940033, -1.0894097089767456], [0.6153764724731445, 1.0868556499481201, -0.2481706142425537, -0.6618127822875977, 1.988539218902588, 0.6259974837303162, -0.45108547806739807, 0.5370092988014221, -1.1196597814559937, 0.5188748240470886, -0.27003055810928345, 0.30138954520225525, 0.9521999955177307, 1.3660988807678223, 0.3131972849369049, -1.280059814453125, -0.1556323915719986, -0.10835893452167511, 1.4022672176361084, -0.2044525295495987, -1.724107027053833, 0.3603570759296417, 0.7937347888946533, 0.4855286777019501, 0.9661967158317566, -0.8729512095451355, -0.33209314942359924, 0.052945587784051895, 0.7438967823982239, -2.1625747680664062, -1.1396760940551758, 0.3377910256385803, 0.3491919934749603, 0.2001948207616806, 0.8206318616867065, 0.7627831101417542, 0.4127857983112335, -0.5830133557319641, 1.2374722957611084, 1.0900599956512451, 0.7722899317741394, 0.7773447036743164, -0.9289866089820862, -2.1740026473999023, 0.5729042291641235, 0.5920097827911377, 0.06574681401252747, 1.5445886850357056, 0.23569226264953613, -1.2494962215423584, -0.2819366455078125, -1.6197700500488281, 0.3628523349761963, 0.5539100766181946, -0.17380934953689575, 0.17030169069766998, -0.2959093153476715, -0.04768754541873932, -3.8590617179870605, -0.35721415281295776, 0.45953038334846497, -0.9533089399337769, 0.5862516164779663, -1.7546701431274414], [-1.7317386865615845, -0.22350262105464935, 1.2103159427642822, -2.2624940872192383, -1.717390537261963, -0.4235154986381531, 0.20190759003162384, -0.020718930289149284, -1.3133399486541748, 1.5688540935516357, -0.30657637119293213, 0.45364058017730713, -2.68925142288208, -0.35843372344970703, 0.7740474343299866, -0.5424839854240417, -0.5628905296325684, 0.48728975653648376, 0.07588937133550644, -0.3306383788585663, -0.6502680778503418, 0.058740075677633286, 1.0043752193450928, -0.4523963928222656, 1.122253656387329, 1.4152305126190186, -1.6552172899246216, 1.0640572309494019, -0.22860176861286163, 0.7608645558357239, 0.5472070574760437, -1.215620994567871, 0.41500839591026306, 0.9544509053230286, 0.09048708528280258, 0.9534882307052612, -0.540785551071167, 0.5529364943504333, 0.4727696478366852, -0.22580960392951965, -2.4910013675689697, -0.42692631483078003, 0.8201211094856262, 1.0934628248214722, -0.04653143137693405, 0.08105144649744034, 0.5686443448066711, 0.14574101567268372, 0.05369298905134201, 0.15361912548542023, 1.2803332805633545, -1.2180004119873047, 0.09662234038114548, 0.07663723826408386, -0.45533716678619385, 0.7375157475471497, 0.606999933719635, 0.2961002588272095, -1.4681812524795532, 1.5051772594451904, -1.4740017652511597, 1.5091190338134766, 0.3786093294620514, -0.7627265453338623], [0.1478300243616104, 0.48137354850769043, 1.6040501594543457, -0.38593965768814087, -1.2221893072128296, -0.6648731827735901, -0.6078935861587524, -0.8022149205207825, -1.1294076442718506, 0.43647778034210205, 1.2807118892669678, 0.8358514308929443, -0.27009183168411255, -0.1428428441286087, 0.680992603302002, -0.8008005619049072, 0.20022056996822357, 0.19303753972053528, -0.8837168216705322, -1.2390803098678589, -0.20163322985172272, 0.21183548867702484, -0.9963406920433044, -0.4044646620750427, -2.3309261798858643, -0.11761826276779175, 0.46672743558883667, -0.21514862775802612, -0.4764803349971771, -0.1316692978143692, -2.104827404022217, 0.03831445053219795, 0.5213579535484314, 1.1037187576293945, -0.07273630052804947, -0.27293986082077026, 0.6934176683425903, 1.7148029804229736, 2.1612119674682617, 1.788379430770874, 0.6471500396728516, 0.5904057621955872, -0.6798675060272217, 0.7566239833831787, -1.2195549011230469, 1.1651160717010498, -0.3979141414165497, -0.07446540892124176, -1.4414644241333008, -0.47098398208618164, 0.7630616426467896, 0.6575285792350769, 1.0701422691345215, 0.9036473631858826, -0.34307989478111267, 0.8215106725692749, 1.9105596542358398, -0.00775570422410965, 0.9172684550285339, 0.38606199622154236, 0.7073220610618591, 1.1302109956741333, 1.4502336978912354, 0.21720004081726074], [0.3922080993652344, 1.7328544855117798, -1.195819616317749, 1.034652590751648, -1.7202004194259644, -0.16378436982631683, 0.25459688901901245, -1.2977651357650757, -2.2334184646606445, 0.6260671615600586, 0.715468168258667, 0.951758623123169, 0.27550527453422546, -1.1996817588806152, 0.2685186564922333, 0.37736162543296814, 1.0202001333236694, -0.8628032207489014, -0.0024503315798938274, -0.8816697001457214, 0.8546812534332275, -0.39316755533218384, 1.1131566762924194, 1.6154162883758545, -0.11814049631357193, 0.7791730165481567, -0.0399925597012043, 0.3378649055957794, -0.1960223764181137, 1.5341745615005493, -1.4346011877059937, 1.4853345155715942, 0.06598204374313354, -0.38175714015960693, 0.24475106596946716, -1.3513264656066895, -0.5411115288734436, 1.0379151105880737, 0.3501952588558197, 0.09899559617042542, -1.141870379447937, 0.5403164625167847, 2.4086732864379883, 0.45867353677749634, 0.6703604459762573, -0.7550755143165588, 1.4262206554412842, -1.4545239210128784, -1.2676634788513184, -0.7333465814590454, -0.5612706542015076, -1.0454776287078857, 1.4868296384811401, 0.30558618903160095, -0.2805836796760559, -1.209472894668579, 0.4411235451698303, -0.5310412645339966, 1.5803183317184448, -0.25878605246543884, -0.17984126508235931, -0.9255642890930176, 1.1221082210540771, -1.1196036338806152], [-0.8394775986671448, -0.1368998885154724, -0.9525729417800903, -0.7765422463417053, -0.15709498524665833, -1.1542844772338867, -1.2656713724136353, 0.7010051012039185, 0.8532281517982483, -1.305243730545044, -0.034734319895505905, -0.4527648389339447, -0.7718806266784668, 1.778091311454773, -0.878187894821167, -0.5490555167198181, -0.49795448780059814, 0.6548342704772949, -0.597373902797699, 1.3172117471694946, -1.2422943115234375, 0.17802003026008606, 1.4616948366165161, 1.0813356637954712, 1.0710731744766235, -1.1021945476531982, -0.7046601176261902, -0.1993779093027115, -0.27870476245880127, -0.001043403521180153, -0.9278799891471863, 1.4383326768875122, -1.213232398033142, 1.5144621133804321, 1.4356250762939453, 0.21792452037334442, 0.019991585984826088, 0.5061284303665161, -0.10170398652553558, 0.9145315885543823, -0.5584173202514648, 0.015946969389915466, -0.9354318976402283, -0.28368625044822693, -0.32303735613822937, -0.6623623967170715, -0.20042139291763306, 0.20706568658351898, -0.19486624002456665, 1.4094096422195435, 3.3718693256378174, 1.2481764554977417, 0.22111138701438904, 1.07722008228302, -1.9077012538909912, 0.1606273353099823, 0.4453478455543518, 0.6571409702301025, -0.04187930002808571, -0.5695359706878662, -0.34069427847862244, -0.776772677898407, -0.6121805310249329, -0.021703604608774185], [1.5601376295089722, -1.326372504234314, 0.8012239933013916, -2.41685152053833, -0.9017706513404846, -1.3843662738800049, 1.7097983360290527, -0.011872464790940285, 2.0356316566467285, 0.9866892099380493, -1.0318933725357056, -0.46709945797920227, 1.0365456342697144, 0.5907675623893738, -0.8019011616706848, -1.8550032377243042, 0.4708702862262726, 0.45950794219970703, -0.07333488017320633, -0.08391077816486359, 0.11727878451347351, 0.6186257600784302, 0.32570239901542664, -0.14011722803115845, 1.2428479194641113, 1.6674022674560547, -1.697534441947937, 0.27681922912597656, 1.1988266706466675, 2.557687282562256, -0.16411270201206207, -1.7747727632522583, 1.9052022695541382, -1.2822909355163574, 0.7433834671974182, 1.6082773208618164, 0.8718065023422241, 0.16438674926757812, 0.1713133156299591, 1.061200499534607, 0.6593005657196045, -0.30012768507003784, -2.830913543701172, -0.20967769622802734, 0.3572428524494171, -0.2917194664478302, 0.2356683909893036, 0.9138140082359314, -0.9720609784126282, -1.3281677961349487, -1.5609380006790161, -1.262061357498169, 0.6053693890571594, -0.07341843843460083, 0.40844956040382385, -1.4156841039657593, -1.3582193851470947, -1.670924425125122, -0.4327588975429535, -0.332886666059494, -1.6103092432022095, -0.4950997233390808, -2.672149658203125, 0.30154141783714294], [0.31263402104377747, -0.42250415682792664, -0.28437119722366333, -0.028606103733181953, -0.819226861000061, 0.7512073516845703, 1.7093816995620728, 0.17648082971572876, -0.3747333586215973, -0.4902399778366089, 2.565218210220337, -0.3885583281517029, -0.8062934279441833, 0.8140856027603149, -0.219784677028656, -0.09725907444953918, 0.9644399285316467, 0.3949207365512848, -0.8626537919044495, -0.6357897520065308, -2.588381052017212, 1.2349183559417725, 1.6068947315216064, 0.6786167025566101, -0.9891711473464966, 0.31605589389801025, -1.832732915878296, 0.14379949867725372, 0.5136747360229492, 0.04209493100643158, -1.4339942932128906, 1.513085961341858, 0.8118886351585388, -0.14289405941963196, -0.6514171957969666, -0.4282362461090088, -0.22059522569179535, -0.23880571126937866, -1.151850938796997, -0.5393129587173462, -0.13805606961250305, 0.354415625333786, 1.3322702646255493, -0.5482897758483887, -0.0842800959944725, -1.8586324453353882, -0.0035527870059013367, 0.2602565884590149, -0.07820269465446472, 0.34546422958374023, -1.5562255382537842, -0.3801417052745819, -0.5206905603408813, -1.4991530179977417, 0.5261363387107849, -0.18976052105426788, -0.949494481086731, 0.7577828764915466, 0.231284961104393, -0.9073081612586975, -0.8139470219612122, -1.406122088432312, 0.6671257615089417, 1.5252196788787842], [0.7808303236961365, 0.3582460582256317, -2.28483247756958, -0.597842812538147, -1.1268434524536133, 0.836247980594635, 0.6525046825408936, -1.2655161619186401, 0.10477551817893982, -2.8024985790252686, -0.06769339740276337, 1.8265135288238525, -0.7720077037811279, -0.7934175133705139, -0.20978273451328278, -0.07712046056985855, 0.40122273564338684, 2.244842767715454, 0.07076392322778702, 0.43506497144699097, -0.050766896456480026, -2.122800588607788, -0.7228533029556274, 0.9798071980476379, -1.1202290058135986, 0.5229353308677673, 0.8586434125900269, -0.6768523454666138, -1.1586024761199951, -0.9007149934768677, 0.9186227917671204, 1.0541017055511475, -0.6838567852973938, 0.8790399432182312, 0.5948448181152344, -0.05588092654943466, 2.239845037460327, -2.3852310180664062, 1.4723068475723267, -0.8276917934417725, -0.29152148962020874, -0.27462825179100037, 1.8213942050933838, 1.5593576431274414, 0.06615707278251648, -0.501465916633606, 1.12225341796875, 0.8873863220214844, -0.09601393342018127, 0.9140487909317017, -0.397065669298172, 0.7831048965454102, -0.2862822115421295, 0.5250087380409241, 1.2956748008728027, 0.5593767166137695, -0.24582208693027496, 0.36023977398872375, -0.9380941390991211, -0.9576114416122437, -0.31145521998405457, -0.9319427013397217, -1.069278359413147, 0.33409860730171204], [-2.1975255012512207, 0.12275394052267075, 1.2530955076217651, 0.8569879531860352, 1.6045130491256714, -0.9441651105880737, 0.06503462791442871, 0.15405137836933136, -1.0603182315826416, -1.3522142171859741, -0.015013680793344975, 0.05915401130914688, -0.19124294817447662, -0.11070354282855988, -1.7964067459106445, -1.900537371635437, 0.23809881508350372, -0.6419126987457275, -1.709176778793335, -0.9156942963600159, -0.7478592991828918, -0.03914061188697815, -1.3601763248443604, 1.2401312589645386, -1.2989966869354248, 1.5869923830032349, 0.571026623249054, 0.31477028131484985, -0.7042003273963928, -0.2695488929748535, -0.18471890687942505, -0.7447754144668579, 1.4274420738220215, -0.6862927079200745, 1.364263892173767, -0.7857643365859985, 1.2001320123672485, -1.9134291410446167, 1.2632280588150024, 0.7277145385742188, -0.7762183547019958, 0.06842155009508133, -0.8271470665931702, -0.5985360145568848, -0.9060459136962891, -0.7655948400497437, 1.3771119117736816, -1.189464807510376, -0.06173265352845192, -1.4289950132369995, 0.10489404201507568, -1.5827761888504028, -0.2992066740989685, -0.5267724394798279, 1.317621111869812, -0.5576159358024597, 0.043852437287569046, 0.1447419822216034, -1.149827003479004, 0.8350362181663513, 0.9354942440986633, 1.4923985004425049, 1.4878795146942139, 0.9022555351257324], [-1.4614862203598022, 0.8945724964141846, 0.5451592803001404, -1.2772027254104614, -0.43577396869659424, 0.5726839303970337, -1.6341660022735596, -0.24776428937911987, 0.4678979516029358, -0.1668146699666977, -0.7894983887672424, -0.31686970591545105, 0.5462737083435059, -0.27191442251205444, 1.4225887060165405, 0.45108404755592346, 0.2905615270137787, -0.44830241799354553, 0.7514897584915161, -0.036800723522901535, 2.426088333129883, -0.07103526592254639, 0.31370967626571655, -1.161636233329773, 0.9142085313796997, -0.25516802072525024, 1.771278738975525, -0.10135043412446976, 0.984394371509552, -0.1247391551733017, -0.5696288347244263, 0.3220725655555725, -1.3886770009994507, 0.36800408363342285, 1.234586238861084, -1.3085623979568481, 1.4781064987182617, 0.974926233291626, 0.019549887627363205, 1.5949633121490479, -1.203831434249878, -0.43549612164497375, -0.43757742643356323, -0.7175471782684326, 0.2846463620662689, 1.2957649230957031, 1.9391578435897827, 1.7156323194503784, 0.14731436967849731, -0.592292845249176, -0.786266028881073, 1.2152321338653564, -0.9769077897071838, -0.2837398052215576, 1.0663020610809326, -1.3457354307174683, -1.2648500204086304, -0.383947491645813, -0.6164470911026001, 1.2240533828735352, -1.144444227218628, -0.2578883171081543, 1.2161102294921875, -1.6314092874526978], [1.8687363862991333, -0.8241686820983887, 0.17259356379508972, 1.9386651515960693, 0.18892152607440948, 0.21430839598178864, -0.8425931334495544, -0.26147180795669556, -0.07074681669473648, -0.49433720111846924, 1.3057241439819336, 1.5143225193023682, -0.2919960916042328, 0.3831259310245514, 0.056828293949365616, -0.3496476113796234, -0.7734793424606323, -0.7246650457382202, 0.6409170031547546, -0.15889672935009003, -0.5356665253639221, -0.470028281211853, -2.5117380619049072, -0.6574183106422424, 0.8524627685546875, 0.8834115266799927, 0.6065446734428406, -0.19117753207683563, 2.274000406265259, -0.16044780611991882, 0.09322943538427353, -0.2788442373275757, -0.578287661075592, -0.01823517307639122, 1.288625717163086, -0.8462479710578918, -1.08377206325531, 1.4601359367370605, -1.3224371671676636, -0.7334136962890625, -2.286893844604492, 0.6258589029312134, 0.6422197222709656, -0.6742197275161743, 1.1426339149475098, -0.2943297326564789, 3.6903295516967773, 0.6364061832427979, -0.5697466135025024, 0.8397097587585449, 0.0028150530997663736, 0.7504761815071106, 2.1823718547821045, 0.31423047184944153, -0.36048561334609985, 0.45617592334747314, -2.1658616065979004, 1.1277409791946411, -0.4380859434604645, -0.8503888845443726, -0.3636651933193207, -1.0474954843521118, -1.1205615997314453, -1.8686082363128662], [0.34551429748535156, 1.9217623472213745, 0.0014165929751470685, 0.5172044038772583, 1.0557641983032227, -0.10090088099241257, -0.6386339664459229, 0.6176192760467529, -1.0548245906829834, 1.0431092977523804, 1.0507088899612427, -0.41348356008529663, 1.0876574516296387, 1.132517695426941, -1.0626616477966309, 0.7711400389671326, -1.6552326679229736, 0.1710236817598343, 0.11579015105962753, 0.14119817316532135, 1.0213569402694702, 1.536324381828308, 0.2970196604728699, -0.9133610725402832, 0.5667347311973572, 1.2818323373794556, -1.0495160818099976, -0.4939916729927063, 0.8605467677116394, 0.8176056146621704, -2.438375234603882, -0.41027581691741943, 0.508859395980835, 0.616967499256134, -1.152038812637329, -0.04559613764286041, -0.6465647220611572, 0.9525930285453796, 1.273936152458191, -0.5954148769378662, -0.6382551193237305, -1.1712679862976074, -0.15859882533550262, -0.06639761477708817, -0.6082611680030823, 1.2510377168655396, 0.6505621075630188, 0.6390470862388611, 1.5942749977111816, -0.565065324306488, -1.044812798500061, 0.37636885046958923, 0.8027441501617432, 0.618466854095459, 1.5823496580123901, 0.5431661009788513, -0.6726440191268921, 0.8878702521324158, -0.12055958062410355, -0.5272479057312012, -2.767355442047119, 2.4595210552215576, 1.265878677368164, -1.37478768825531], [-0.2473730891942978, 0.1900063008069992, 0.03933078423142433, -0.580578088760376, -1.2341490983963013, -0.6727824807167053, -0.8194623589515686, 0.2257222682237625, -1.6120431423187256, 0.9889333844184875, -1.2607909440994263, 0.5632310509681702, -0.05042167752981186, 1.457822561264038, -1.3189691305160522, -0.3226904273033142, 1.569707989692688, -0.6028758883476257, -0.28724732995033264, -0.2020559012889862, -0.8624286651611328, -0.3211512565612793, -1.0946177244186401, 0.7459890842437744, 0.09583818912506104, -0.018881922587752342, -0.5028078556060791, -0.49206626415252686, -0.46224772930145264, -0.8291584849357605, -0.523704469203949, 0.9633015990257263, -0.6881391406059265, 0.22032025456428528, 1.1367912292480469, -0.03476901352405548, -0.22424490749835968, 0.6039150357246399, -1.7759885787963867, 0.692206084728241, 0.18357931077480316, -0.14264948666095734, -0.04106221720576286, 1.4595402479171753, 0.9467179179191589, 0.7805822491645813, -0.317702054977417, 0.4947572350502014, 1.6081887483596802, 2.013047456741333, 1.2047191858291626, -0.07489190250635147, 1.5610482692718506, 0.48217958211898804, 0.3692431151866913, -0.8989855051040649, -0.565226137638092, 0.3447103202342987, -2.2928898334503174, 1.2513939142227173, 1.092438817024231, 1.1389118432998657, 0.8147364854812622, -0.06198160722851753], [1.5150761604309082, 0.49134835600852966, 0.9032646417617798, 0.04194368049502373, -0.7494686841964722, 0.4172731041908264, -0.7163192629814148, -0.9094721674919128, -0.8185374736785889, -0.22653615474700928, -2.0698113441467285, -0.07530705630779266, 1.503093957901001, 1.174439549446106, -1.850180983543396, 0.9063839912414551, 0.5560895800590515, 0.5575008988380432, 0.39717328548431396, -0.4057217240333557, -1.4340065717697144, 1.8930710554122925, -0.444813996553421, -0.314866304397583, -0.979520857334137, -0.42869263887405396, 0.07345219701528549, -0.25357601046562195, -1.2239168882369995, -0.7106541395187378, -0.16243697702884674, -1.2134495973587036, -1.0009880065917969, -1.846947193145752, 0.4246215224266052, -0.16151097416877747, -0.8473290801048279, 0.39805665612220764, -0.44340580701828003, -0.9007582664489746, 1.4250785112380981, -2.1174118518829346, -1.520182728767395, 0.14279355108737946, 0.02765250951051712, -0.8866646885871887, 0.4160136878490448, 0.901011049747467, 0.28895145654678345, 0.14562512934207916, -1.7963922023773193, -1.1678745746612549, -0.09536829590797424, 0.41723203659057617, 1.1033658981323242, 0.3737371265888214, -0.7511450052261353, -0.3154255449771881, 0.844420313835144, 2.2650492191314697, 0.24755553901195526, 1.3902674913406372, -0.6843528747558594, 0.2524724006652832], [0.4937695860862732, 0.5592615604400635, 0.4645403325557709, -0.7802494168281555, -0.5872524380683899, 1.574573278427124, -1.929297924041748, 0.731997549533844, 0.31219425797462463, -0.695157527923584, -1.5367640256881714, -0.5464417934417725, -0.28254058957099915, -0.057814255356788635, 0.4350351095199585, -0.05724272131919861, 1.0427805185317993, -1.3498703241348267, -1.0115232467651367, -1.6431386470794678, -0.939120352268219, 0.5435917377471924, 0.3159044682979584, -0.8187789916992188, -0.3115370571613312, -0.23170775175094604, 1.920459270477295, -0.38509753346443176, -0.8037963509559631, -0.09518393874168396, -1.5851751565933228, -0.418349951505661, -0.41986918449401855, -0.623564600944519, 0.7439906001091003, -0.5713241696357727, -0.16271314024925232, 1.037349820137024, 1.574745535850525, 1.3113330602645874, -0.2652231454849243, -0.38018789887428284, 0.4392613172531128, 1.387657642364502, 1.689314603805542, 0.32646888494491577, 0.5956293940544128, 0.41435006260871887, -1.0517040491104126, -0.9608669877052307, -0.7075677514076233, 0.8035176992416382, 0.36666980385780334, 2.198819160461426, 1.5961949825286865, -0.21954557299613953, -0.777439534664154, -1.2702388763427734, -0.976560652256012, -0.22551164031028748, -0.29520484805107117, -1.3825424909591675, 1.105675458908081, 2.1971988677978516], [0.015775378793478012, -1.7469236850738525, 0.2667187452316284, -0.11162545531988144, 0.6329524517059326, -0.6655957102775574, -0.1417643427848816, -1.193576693534851, -0.39979875087738037, 0.5231906175613403, 0.07175145298242569, -0.5875439643859863, 0.19100603461265564, -0.923316240310669, -2.058760643005371, -1.6656304597854614, 0.6963074803352356, -0.6972441077232361, 0.10978183150291443, 0.7336323261260986, 0.13704736530780792, -0.38837406039237976, 2.0692834854125977, -0.21711789071559906, -0.9185253977775574, 0.24306464195251465, -0.012627068907022476, -1.0724201202392578, 0.046126969158649445, -2.173199415206909, -0.3664740025997162, -0.5260770320892334, -1.8420381546020508, 0.9582314491271973, 1.0076113939285278, -0.22034436464309692, 0.546818196773529, -0.6173995733261108, 0.8535854816436768, -0.005822306964546442, -0.5518780946731567, 1.173593282699585, -1.1824204921722412, -0.31361863017082214, 0.17114175856113434, -0.04930059239268303, 1.0979679822921753, 1.219169020652771, -0.47114044427871704, -1.254827618598938, 0.11659038811922073, -0.7791606783866882, 0.2926919460296631, 1.7558586597442627, 0.6196498870849609, -0.4912824332714081, 0.6309790015220642, 0.4090110957622528, 1.6077854633331299, 0.7016309499740601, 0.6743454933166504, 0.5754101276397705, 1.0122584104537964, -2.182488441467285], [0.9850561022758484, -0.37525132298469543, 2.735994577407837, 0.5478523969650269, -0.9915621280670166, -0.08851465582847595, 0.6627570390701294, 1.39802885055542, -0.644697904586792, 0.28376784920692444, -0.35393545031547546, -0.18430018424987793, -1.0549002885818481, 1.2122836112976074, -0.35107773542404175, -0.3591642379760742, -0.586761474609375, -1.5477213859558105, -0.6127898693084717, 0.6892269849777222, -1.4977971315383911, 2.0550272464752197, 0.6565585136413574, -0.34683722257614136, -0.8120576739311218, -0.878845751285553, -0.06344297528266907, -1.7536041736602783, 0.8335961699485779, 0.23324184119701385, 0.5807284116744995, -0.4524228274822235, -1.1079474687576294, 2.049320936203003, 2.16982102394104, 0.3777077794075012, -0.45203423500061035, -0.8488128781318665, 0.28161782026290894, 2.023237705230713, 1.5857630968093872, -0.6371037364006042, -0.8317217826843262, 0.8475268483161926, 0.9532213807106018, -0.30582547187805176, -1.4527405500411987, 1.8766603469848633, -0.5750434398651123, -1.9672030210494995, -0.5669257640838623, 0.3455553948879242, -1.191368818283081, -1.7392221689224243, 1.5012816190719604, -1.4884827136993408, 1.163770079612732, -0.07403048127889633, -0.8986312747001648, 0.8916926383972168, 1.0557072162628174, 0.21849587559700012, 1.7093524932861328, 0.2687649726867676], [-0.1271515190601349, 0.6887381672859192, -1.3841420412063599, -0.2950586676597595, 0.5125404596328735, 0.8372841477394104, 1.4341890811920166, -1.1675969362258911, 0.5095304846763611, 0.16482429206371307, -0.15426252782344818, -0.22381356358528137, -1.058104395866394, -0.04666799679398537, -0.3138754963874817, -1.476032018661499, 0.11948812007904053, -0.8601469397544861, 0.11320246756076813, 2.4721767902374268, -0.8313823342323303, -0.6047226190567017, -1.2795226573944092, 1.4370805025100708, -3.2699697017669678, 1.1827720403671265, -0.6154451966285706, 1.5073342323303223, 0.537146806716919, -0.3786346912384033, -0.47372007369995117, 0.8654365539550781, -1.551261305809021, 0.3617057204246521, -0.09389922022819519, 0.6954514980316162, 1.0252605676651, -0.1753932386636734, 1.212207317352295, 1.4647616147994995, -1.0289902687072754, 0.41312316060066223, -0.020227668806910515, 1.7790507078170776, 0.9178483486175537, -0.04467979446053505, 0.38320425152778625, -1.4870871305465698, -0.8785273432731628, -2.046588182449341, 0.5406541228294373, 0.5487181544303894, 1.265214204788208, -0.06879927963018417, 1.2911574840545654, 1.599113941192627, -0.08833932131528854, 0.19085881114006042, -0.3834259808063507, -1.0063554048538208, 0.4781032204627991, 1.4115352630615234, 0.1518663614988327, -0.2076353132724762], [-1.1857025623321533, 0.8211613893508911, -0.6756325960159302, -0.08940819650888443, 0.1724715381860733, -0.8119901418685913, -1.1322671175003052, -0.7460334300994873, 0.2684308588504791, 0.40760537981987, -0.81437748670578, -0.19817067682743073, 0.5826569199562073, -1.8646502494812012, 1.355720043182373, -1.1642589569091797, 1.263983964920044, 0.8018724918365479, -0.18278996646404266, 0.9765899777412415, 0.16599580645561218, 0.07668063789606094, 0.6710880994796753, -2.311168670654297, 1.9113335609436035, 1.5630943775177002, 1.380649209022522, 0.4804288446903229, 0.6692016124725342, -1.0998988151550293, -0.7070251703262329, 1.0429593324661255, 1.0809179544448853, -1.221436858177185, 0.9827450513839722, 0.9851322174072266, -0.7448632121086121, 0.46621057391166687, -0.20435857772827148, 0.4376467168331146, -0.9976141452789307, 1.0000531673431396, 0.014135106466710567, 0.4934273362159729, 1.0741260051727295, 1.4074105024337769, 0.6517670750617981, 1.3244397640228271, -0.13639475405216217, -0.19300717115402222, -0.06890533119440079, 0.401336133480072, 0.5525804162025452, 0.3381003141403198, -0.7269446849822998, 0.35457250475883484, -0.3506997525691986, 1.3357441425323486, 0.14079108834266663, 0.13871566951274872, -0.8988986611366272, -0.03525641933083534, -0.8036153316497803, 2.1869471073150635], [-0.9383803606033325, -0.4301510453224182, -0.4534730911254883, -0.6402871012687683, -1.653505802154541, -0.099424809217453, 0.19347597658634186, -1.6712042093276978, -2.0687570571899414, 0.005172271281480789, -0.316228985786438, 1.0456758737564087, -0.2752663195133209, -0.3626970946788788, -0.9159096479415894, -0.5878442525863647, -0.4915674030780792, -0.10413195192813873, 0.37839215993881226, 1.1744277477264404, 1.771369457244873, 0.49199607968330383, 0.7357069849967957, 0.2719576954841614, 0.5667021870613098, -1.1904809474945068, -1.1810669898986816, -0.18652202188968658, 0.06548528373241425, 1.044026494026184, 0.2360290288925171, -0.4078652560710907, 1.1578645706176758, 0.2129957526922226, -0.1899944245815277, 0.4240199327468872, 1.0176516771316528, -0.4302988052368164, -1.0034674406051636, 0.9520655870437622, 1.6842334270477295, -0.24696099758148193, 2.016435384750366, 2.021221160888672, 0.39350807666778564, 1.2138103246688843, -1.0705184936523438, 1.9238849878311157, -2.7076711654663086, -2.067817449569702, -0.21583223342895508, 1.729964256286621, -0.535136878490448, 1.9803816080093384, 0.3685290217399597, -1.0493818521499634, 0.6279047131538391, 0.9386867880821228, -0.46282872557640076, 0.5923480987548828, -0.29541638493537903, 0.1977986842393875, -1.015931248664856, -0.6274994015693665], [-0.08045989274978638, -0.5932252407073975, 1.2742257118225098, 0.8132404088973999, 0.34574422240257263, 0.01435284223407507, 0.6071338057518005, 0.29557546973228455, 0.07270567119121552, 0.6871446967124939, 1.966198205947876, 0.9113029837608337, 0.6960258483886719, -0.5247185826301575, -0.9442672729492188, -1.4684079885482788, -1.8404361009597778, -0.9996240735054016, -0.3970465362071991, -0.04805689677596092, 0.08378449827432632, -0.2597576677799225, -1.9925090074539185, -1.2168667316436768, 1.5375009775161743, 0.030186763033270836, 0.6421254873275757, 0.18035733699798584, 0.37172433733940125, -0.48036885261535645, 0.05306031554937363, -0.7684449553489685, -2.1038506031036377, -0.6039093732833862, -0.5753942131996155, 0.031706780195236206, 1.6740857362747192, 0.968638002872467, -2.3435797691345215, -0.0156497061252594, 1.6860519647598267, -0.07225792855024338, -0.5766288042068481, 1.492344617843628, -2.304366111755371, -0.37701624631881714, -0.3958369493484497, -1.8088529109954834, -0.05664711073040962, 0.14013433456420898, 0.3766135573387146, 0.57347571849823, 2.002768039703369, -1.3007429838180542, -0.8859502077102661, -1.6824172735214233, 0.2362629771232605, -0.8491218686103821, 0.03628337383270264, -0.5231238007545471, -1.706309199333191, 0.9122947454452515, 0.7105284333229065, 0.13565057516098022], [-0.7859114408493042, -0.23712581396102905, -0.5659710764884949, 0.021389182657003403, -0.17800113558769226, 0.10579486936330795, 0.7213566899299622, 1.0833849906921387, 0.06685885041952133, 1.459263563156128, 1.0435059070587158, 1.417937994003296, -0.8813304901123047, -0.15923529863357544, -0.6174411773681641, 0.5272821187973022, 0.7572510838508606, -1.2106330394744873, 0.040553249418735504, -1.2773712873458862, 0.8076708316802979, -0.9871882200241089, -0.8748373985290527, -0.8574776649475098, 0.1348244994878769, 2.057121753692627, 1.5526316165924072, -0.30581578612327576, 0.07151565700769424, -1.2399870157241821, 1.9319384098052979, -0.7638084292411804, -0.26111334562301636, -1.1104413270950317, 2.2500109672546387, 0.37860676646232605, -0.42783796787261963, 0.2551986277103424, -1.662140130996704, 0.957595944404602, 1.9594700336456299, -0.34056490659713745, 1.9279085397720337, 0.8351641893386841, -0.17336468398571014, -1.8296688795089722, -0.3598775565624237, 0.07562106102705002, 0.5967483520507812, -1.4894472360610962, 0.5920782685279846, -0.38842397928237915, 0.4257905185222626, 2.008615732192993, -1.124284267425537, 0.5450829863548279, -2.4601190090179443, -0.5933071970939636, -0.8944697380065918, -1.9607138633728027, 1.9201470613479614, -0.8611235022544861, -0.3281501829624176, -0.06621308624744415], [0.5980152487754822, 0.3497483730316162, 1.4409903287887573, 0.051420487463474274, -0.30441948771476746, 0.761444091796875, -2.3910531997680664, 0.18144235014915466, -1.1134260892868042, 1.0117243528366089, 2.765662670135498, 1.0257607698440552, 1.226636528968811, 0.6142398715019226, -0.24065439403057098, -1.3377028703689575, -0.29192259907722473, 0.25879594683647156, -1.0842612981796265, 0.6372902989387512, 0.25144410133361816, 0.13881970942020416, -0.075325146317482, -0.20469702780246735, 0.9052507281303406, -0.6198163032531738, -1.6057664155960083, 1.225926399230957, -1.5269615650177002, 0.49191901087760925, -0.2545468211174011, -1.4060382843017578, -0.1328711360692978, -0.0767967626452446, 0.5184771418571472, 1.1794778108596802, -0.2496791034936905, 1.5362204313278198, 0.5262519717216492, 0.7762206792831421, -1.5506373643875122, -1.0188822746276855, -1.5013772249221802, -2.0971603393554688, 0.5282203555107117, -1.1712757349014282, -1.2160505056381226, 0.19294658303260803, -0.8607947826385498, 0.7587280869483948, 1.5716208219528198, 1.2828186750411987, -0.10788295418024063, -0.8878756165504456, -0.2726200222969055, 0.09097559750080109, -1.298384189605713, 1.2488189935684204, -0.5524731278419495, 0.9968363046646118, -1.5985013246536255, 0.7651483416557312, -0.10652865469455719, 0.6824228763580322], [0.21076732873916626, -0.5115485787391663, 0.025062544271349907, 0.21574467420578003, -0.9554786682128906, -0.07324396818876266, 0.11160224676132202, -0.7797687649726868, -1.3049156665802002, 0.31361573934555054, -0.6413431763648987, 0.18774236738681793, 0.7403220534324646, -0.4608609080314636, 1.7532062530517578, -1.3713433742523193, 0.9466629028320312, -0.9145359992980957, -0.12474171817302704, -0.12468786537647247, -0.8170686364173889, -0.9461532235145569, -1.2079585790634155, 0.6785587072372437, -2.2811830043792725, -0.1545027196407318, 0.11800485849380493, 0.533810019493103, -0.7699174284934998, -1.0619785785675049, -0.4897347092628479, 1.881838083267212, -0.5831609964370728, 0.6942166090011597, 0.7761756181716919, 1.1387218236923218, -0.38701725006103516, -0.16820302605628967, -1.6725481748580933, -0.32819801568984985, -1.3308191299438477, 1.104498267173767, -0.5431277751922607, 2.1002488136291504, 0.39901450276374817, -0.6233335733413696, -0.19735470414161682, -0.37617772817611694, -0.786560595035553, 1.4138239622116089, 0.30182525515556335, 1.117165207862854, 1.461427927017212, -0.10730575025081635, 0.7369381785392761, 0.38218510150909424, 0.733783483505249, -1.3983032703399658, 0.13853994011878967, 1.019385576248169, 1.0971189737319946, 0.18710456788539886, 1.0940159559249878, 1.4811242818832397], [-0.8444156646728516, -1.8434442281723022, -1.112720251083374, -0.2982986867427826, -0.49492591619491577, 1.4785473346710205, 1.0032141208648682, -0.02768152952194214, -0.3615765869617462, -0.709617555141449, 0.3768177628517151, -0.7388649582862854, 0.9418874979019165, 0.681061327457428, 0.613440752029419, 0.9826293587684631, -0.8398651480674744, -0.5574626922607422, 1.6540722846984863, 1.187462568283081, 1.919134497642517, 0.7445127964019775, -0.47567006945610046, 1.4550426006317139, -0.28876835107803345, -1.0017480850219727, 0.5752638578414917, -0.2097454071044922, 1.6483314037322998, -2.5000739097595215, -0.031363848596811295, -0.4372405707836151, 0.13366582989692688, 0.5467472076416016, -2.1194677352905273, -0.47459858655929565, -0.90433669090271, 0.5404199361801147, 0.6778764128684998, 0.044338807463645935, 0.3202618658542633, 2.2269198894500732, -0.976875901222229, 0.19697314500808716, 2.54093074798584, -0.2507322132587433, -0.5147699117660522, 1.1211183071136475, -1.5156962871551514, 1.8786523342132568, -0.30580398440361023, 0.03609263151884079, 0.8854931592941284, -1.1849465370178223, 0.2685723602771759, 0.5657654404640198, -0.47846361994743347, 0.2006119042634964, -1.039970874786377, -0.1695626676082611, -0.8387994170188904, 0.7363131642341614, -0.644507110118866, -0.6342819333076477], [0.45525896549224854, -1.148929238319397, 0.40621769428253174, 0.9435788989067078, -2.072627544403076, 1.4288972616195679, -1.236271858215332, -0.3689653277397156, -1.140594244003296, -0.5253037214279175, 0.520807683467865, 1.0145496129989624, -1.2712907791137695, -1.8965502977371216, -0.18125635385513306, 0.6493250727653503, 0.21892602741718292, 0.6146265864372253, -0.5128419399261475, -0.3679051101207733, 0.340168297290802, 0.9425602555274963, 0.10951008647680283, -2.351243019104004, 0.283816397190094, -0.1851503551006317, 0.3628494143486023, -0.48319393396377563, -0.4507998526096344, -0.9662152528762817, -1.0598082542419434, -1.554292917251587, -0.6029637455940247, 1.3262399435043335, 0.03560565039515495, -0.8292719721794128, -1.3457638025283813, -0.10166282206773758, -0.3813721239566803, 1.3652265071868896, 1.5811504125595093, -0.17734824120998383, 1.1387622356414795, 1.909598469734192, -0.6928279995918274, -0.9857645630836487, 2.524402141571045, -0.09613539278507233, -0.1903110295534134, 0.8136516809463501, -0.833227276802063, 1.0388885736465454, -1.1017792224884033, -1.6614400148391724, 1.5912002325057983, 0.2834504246711731, 0.2652655839920044, 1.4520065784454346, -1.2973946332931519, -0.6527430415153503, 0.021570181474089622, 0.09738647192716599, -0.4118843972682953, -1.055167555809021], [-0.24018612504005432, 0.3178481459617615, 1.3113300800323486, -1.0862334966659546, -1.038016676902771, 1.1112322807312012, 0.39038655161857605, -1.0501420497894287, -0.8286798596382141, -1.2333283424377441, 0.3686101734638214, 0.12700003385543823, 1.2041128873825073, -0.05862534046173096, -0.06706316024065018, 0.4639946520328522, 0.9490141272544861, 0.7917200326919556, 0.38832125067710876, -1.4341497421264648, 1.1684701442718506, -0.21177858114242554, -0.32219237089157104, -0.28047314286231995, -0.5641211867332458, -1.3423751592636108, 0.15834277868270874, -0.6831264495849609, 1.4145026206970215, -0.9724896550178528, 1.12335205078125, 0.5542714595794678, -0.6122046709060669, -1.8082963228225708, 0.008925789967179298, -0.5040377974510193, 0.22251544892787933, 0.6615579724311829, 0.6122148633003235, -0.19402281939983368, -0.47817665338516235, 0.038029324263334274, 0.7939427495002747, 0.2502170503139496, -1.1923755407333374, 0.9834564924240112, -0.34178367257118225, 0.44782596826553345, -0.6172394752502441, 1.574318766593933, -1.4193569421768188, 0.5010694861412048, 1.3517872095108032, 1.3824355602264404, -1.7765573263168335, -0.5450921654701233, -0.8528116941452026, -0.21873383224010468, 0.1597173810005188, 0.3142140805721283, 0.7562843561172485, -0.1423572152853012, -1.5042771100997925, 1.950326919555664], [0.1475811004638672, 0.4976865351200104, 0.1098499447107315, -0.43429920077323914, 2.2227957248687744, -1.3629902601242065, -0.43435195088386536, -0.21902932226657867, -0.4704197645187378, 0.8532772064208984, 0.07403820753097534, -1.0387197732925415, 1.1588451862335205, -0.5091928243637085, -0.6961151957511902, -0.0762413889169693, -0.37751075625419617, 1.6724722385406494, 0.43313947319984436, -2.5198240280151367, -1.115661382675171, -0.5314632654190063, 1.5222296714782715, -1.0942503213882446, 0.9823827147483826, -0.8464526534080505, 0.38976940512657166, 0.6134074926376343, -0.4147471487522125, 0.5771884322166443, -1.2062262296676636, 0.35266080498695374, -0.8591681718826294, -0.27317821979522705, -1.684696078300476, 1.2698583602905273, 0.09050875902175903, 0.576224684715271, 0.6167171001434326, 0.7445028424263, -0.0037728080060333014, -0.87105792760849, 0.2421247959136963, -0.19153685867786407, 0.2661040723323822, 1.1369765996932983, -0.48216181993484497, -0.23602552711963654, 2.5548665523529053, 0.7799162864685059, -0.40064212679862976, 1.2451468706130981, 1.535618782043457, -0.3572438657283783, -1.017295002937317, -0.8570367693901062, -1.6871657371520996, 2.020183801651001, -3.433440685272217, 0.5838896632194519, 1.3348547220230103, 1.0903024673461914, 0.12619109451770782, -1.634995937347412], [0.47816339135169983, -1.1603001356124878, 1.3277192115783691, 1.6297181844711304, -0.14409440755844116, 0.43387919664382935, -0.08517137169837952, -0.2479662448167801, 1.4011307954788208, 1.0347678661346436, 0.36253800988197327, 1.9908537864685059, 1.7634994983673096, -0.026162082329392433, 0.5349087715148926, -0.981131374835968, 0.3936631381511688, -1.108029842376709, 0.4885269105434418, -0.8621707558631897, 1.6342447996139526, 0.11494265496730804, 0.2884271442890167, -0.2130393236875534, 0.7064094543457031, -1.494827151298523, -0.3371984660625458, -0.5348542928695679, 1.1515698432922363, -0.7841753959655762, 0.15363192558288574, -1.7465167045593262, -0.2977653443813324, 0.04628912732005119, -0.17696908116340637, -0.04047435149550438, -0.5649592876434326, -0.830929696559906, 0.41822850704193115, 1.4443402290344238, -1.8877298831939697, -0.39346960186958313, 1.5111033916473389, -1.2247769832611084, -0.6547141075134277, -0.9539696574211121, 0.05670132860541344, -1.5865380764007568, -0.2360994666814804, 0.3631115257740021, 0.9523975253105164, -1.360235571861267, 1.1274564266204834, -1.0879298448562622, -0.19036976993083954, -0.20884908735752106, 1.4028884172439575, 1.4745838642120361, 0.6694143414497375, -0.8956965208053589, 0.33792001008987427, -0.19217558205127716, 0.3306127190589905, 0.5729268789291382], [1.395642638206482, -0.9309659004211426, 1.890183925628662, -0.4171037971973419, -1.0272951126098633, -0.6949931979179382, -1.5058845281600952, -0.8441245555877686, 1.030049443244934, -0.43580400943756104, -0.4771762192249298, -0.49628475308418274, 0.09127280861139297, -0.35898271203041077, -2.054166555404663, 0.5659449100494385, -0.5324121713638306, -0.496319055557251, 0.35550180077552795, 0.21994416415691376, 1.7025290727615356, -1.304274559020996, -0.9459935426712036, 0.7727683186531067, 0.8048475384712219, 0.7001115679740906, 0.43955081701278687, -2.524545907974243, -0.2706507742404938, -1.2905508279800415, 0.26341482996940613, 0.3396577835083008, -1.2493518590927124, 0.32623687386512756, 0.6183773279190063, -1.6429762840270996, 0.3320278525352478, -1.6684080362319946, 1.790831208229065, 0.09071388095617294, -1.3472687005996704, 0.2857350707054138, -1.253839135169983, 1.5642695426940918, 0.2542913556098938, -0.5695838928222656, -2.242431163787842, -0.08702072501182556, -0.16882829368114471, -0.18864624202251434, 1.3562299013137817, 0.33281567692756653, -0.458863765001297, 2.8012561798095703, 1.4698855876922607, -0.5063738822937012, -0.11987073719501495, -1.3567078113555908, -0.9606993794441223, -1.220417857170105, 1.9273148775100708, 0.15309637784957886, -1.052857518196106, -0.41155901551246643], [0.7238166928291321, 0.6464941501617432, 0.13636896014213562, 0.878282904624939, 1.589031457901001, -0.4738107919692993, 1.139855146408081, -2.1753835678100586, -1.6004023551940918, -0.3530934453010559, 0.878837525844574, 2.0512804985046387, 1.2050549983978271, 0.2597707509994507, -1.3163976669311523, 0.42743372917175293, -1.2975059747695923, -1.0898945331573486, -1.3407591581344604, 0.483227014541626, 1.3162219524383545, 1.4356328248977661, 0.10516440868377686, 1.5433610677719116, 1.011249303817749, -0.382802277803421, 0.6851710677146912, -1.3493553400039673, 0.5023400783538818, -0.22086091339588165, 0.8126561045646667, 0.0037904188502579927, -1.457919716835022, -1.0687669515609741, 0.9822096824645996, 1.7757294178009033, 0.6573851108551025, -1.0634788274765015, -0.48186296224594116, -1.3555055856704712, -0.31444263458251953, 1.522932767868042, 1.5623852014541626, 0.33959031105041504, 1.2688639163970947, -0.8901618719100952, 0.03193262591958046, -0.8049825429916382, -1.9639365673065186, 0.15887628495693207, -1.343661904335022, 1.7998037338256836, 0.8055586218833923, 0.4625837802886963, 0.34705784916877747, 0.1763247847557068, 0.033329639583826065, 0.17453059554100037, -0.06355259567499161, 1.0179815292358398, -0.0029331922996789217, -1.634348750114441, -1.3905258178710938, 0.7938551902770996], [0.6149227619171143, 0.6698266267776489, 1.1242756843566895, 0.37492743134498596, -0.24359895288944244, 0.1225319653749466, -0.1979455053806305, 1.051956295967102, -0.2591571807861328, -0.20725072920322418, 0.6205698251724243, 1.1262513399124146, 0.6072906851768494, 1.04438054561615, 0.6373618245124817, 0.8264414668083191, 1.721476674079895, -0.949172854423523, -0.7842543721199036, -1.0200557708740234, -1.020560622215271, -0.9449020028114319, -0.4376471936702728, -0.7471426725387573, 0.2784549295902252, 1.9597790241241455, 0.05380759388208389, 0.17911241948604584, 1.1415491104125977, 0.9781625866889954, -0.3216494023799896, 0.20315223932266235, -0.39304447174072266, -1.5035207271575928, -0.08697228878736496, 0.4908596873283386, 0.3892609477043152, -0.24928241968154907, 0.7234383225440979, 0.4876355230808258, 0.5654309391975403, -0.13757525384426117, -0.5292494297027588, -0.7358974814414978, 1.113481044769287, -0.22428831458091736, -0.08427412062883377, -0.20688168704509735, -0.23282985389232635, 0.8124313354492188, -0.08007841557264328, 0.7270238399505615, -2.0264792442321777, 0.9346780776977539, -0.7472491264343262, -0.8437389135360718, -0.6322281360626221, -1.2370448112487793, 0.2438998967409134, -0.7777206897735596, -0.7022060751914978, 0.006477469112724066, -1.2264915704727173, 0.9359694719314575], [-0.7388720512390137, -0.7065114974975586, -0.76556795835495, 1.6808115243911743, 0.7335842847824097, 0.0698256567120552, 0.9232298135757446, 0.2685823440551758, 2.3892581462860107, -0.2843319773674011, 0.21992704272270203, 0.19115719199180603, 0.0718357041478157, 1.293570876121521, -0.2757939398288727, 0.7304092645645142, 0.7614848613739014, 0.030138695612549782, -0.21264326572418213, -0.8530499339103699, -1.4212480783462524, 1.3936069011688232, -0.3192187547683716, -0.1622706800699234, 0.9515084028244019, -0.5130364298820496, 0.1556640863418579, -0.41600239276885986, 0.24875247478485107, 1.213224172592163, -2.103600025177002, 1.981553077697754, 1.1302329301834106, -1.0449817180633545, 0.18624693155288696, 0.300251841545105, -0.38526639342308044, 0.8791368007659912, -0.0936746820807457, -0.7484840750694275, 1.0362221002578735, -1.8058782815933228, -0.16497454047203064, 0.1746305674314499, -0.10711448639631271, 0.003174576675519347, 3.882606029510498, 0.8453616499900818, -0.3321000635623932, 2.499770402908325, 1.436171531677246, 1.7790818214416504, 1.850874900817871, -1.0463632345199585, -0.9435563087463379, -0.4865219295024872, 1.3574244976043701, 0.529290497303009, -1.2423092126846313, 0.11919821053743362, 0.6494964957237244, 0.08354870229959488, 0.017917737364768982, -0.604809045791626], [-1.4989389181137085, -0.08435289561748505, -0.2947145700454712, -0.4000372886657715, 0.6540629863739014, 0.227393239736557, -0.9379608035087585, -1.8278495073318481, -0.5769927501678467, 0.29245486855506897, 0.07224412262439728, -0.39985448122024536, 1.2164644002914429, -0.7664613723754883, 0.7406216859817505, -0.016546694561839104, -1.1660977602005005, 0.27964523434638977, 0.0022961890790611506, -0.9560704827308655, 0.9894143342971802, 0.7699236273765564, -0.6395856142044067, 0.9080671072006226, 1.0253496170043945, -0.04590733349323273, 0.8070362210273743, -0.5466004014015198, 2.4000256061553955, -1.5357095003128052, 1.4118931293487549, -0.5091565847396851, -0.216448113322258, -0.40347036719322205, -0.5763200521469116, -0.43794092535972595, -1.0113605260849, -1.7458784580230713, 1.474160075187683, 0.33104148507118225, 0.35930493474006653, 0.63791823387146, 0.5422529578208923, -1.5800225734710693, 0.30108851194381714, -0.5509698390960693, 1.0932128429412842, -1.3450254201889038, -1.3268511295318604, 2.229470729827881, 2.053286075592041, 0.4061146676540375, -0.8316864371299744, 0.18765102326869965, -0.5862047076225281, 1.5475155115127563, -0.3142797350883484, 0.04670051857829094, -0.10524922609329224, -0.8122432827949524, -0.3865835964679718, -1.1118066310882568, -0.9936570525169373, -1.8594928979873657], [0.6161951422691345, -0.026570383459329605, -1.5939677953720093, -0.031432878226041794, -1.4228861331939697, 0.4091651737689972, 1.5594089031219482, -0.6257731318473816, -0.3256990611553192, 0.9251370429992676, -1.1664776802062988, -0.5639909505844116, -0.6648552417755127, 0.24807831645011902, -0.4920627474784851, 0.3793635368347168, 1.3098852634429932, 0.9831200242042542, -0.8047555088996887, 1.9541488885879517, 0.09900364279747009, -0.22376735508441925, -1.719890832901001, -0.19136859476566315, -0.018628139048814774, -0.6129365563392639, -0.7610649466514587, 0.988700807094574, -1.3869516849517822, 0.013518450781702995, -1.1126372814178467, -1.480724573135376, 1.278376579284668, 0.4311830401420593, 0.42035582661628723, 0.8381149768829346, -1.5576746463775635, -0.6064791679382324, -1.1907858848571777, 1.0220698118209839, -4.6608123779296875, 0.1201164722442627, 0.45622575283050537, -1.2859030961990356, 0.8940225839614868, 0.5413160920143127, 0.3758282959461212, 0.8904625177383423, -1.7991317510604858, -0.2003590315580368, -1.359020709991455, -0.16977305710315704, -0.9160766005516052, 2.4015331268310547, -1.7539167404174805, 1.3299751281738281, -0.9872338771820068, 0.4047304391860962, -0.6733329892158508, -0.11078637093305588, 1.0989351272583008, -1.3652918338775635, 0.3827269673347473, -0.5116956233978271], [0.1187763586640358, 0.5090056657791138, 0.7353569865226746, 0.7509030699729919, -0.08204008638858795, -0.6102283596992493, 0.1423856019973755, -0.15637430548667908, 0.7452293038368225, -0.943595290184021, -0.03831912577152252, -1.2088655233383179, 0.22374866902828217, 0.037667591124773026, 0.572555661201477, -0.6229541301727295, -0.2951638400554657, -0.585953414440155, 0.9797989130020142, -0.8111761212348938, 0.8890222907066345, 0.15886613726615906, 1.392574667930603, -1.5087894201278687, 0.5786123275756836, 0.9266610145568848, 0.7037290334701538, -1.416968822479248, 0.20770542323589325, 0.031684208661317825, -1.2646032571792603, -0.44587722420692444, -0.3998766243457794, 1.557004690170288, -1.432006597518921, -0.6982873678207397, -0.6456872820854187, -0.47266000509262085, 1.0145174264907837, -1.4860496520996094, -0.8225609660148621, -0.8873566389083862, -0.6260111331939697, 0.19058091938495636, 0.27446621656417847, -0.3791726529598236, 1.7077816724777222, -0.530913770198822, 0.6176647543907166, -0.8871704339981079, 2.9051432609558105, -0.3723127841949463, -1.454306721687317, -1.7726563215255737, -0.8510837554931641, 0.3036346137523651, 2.8307995796203613, -0.32500505447387695, 1.5125837326049805, -0.02811616286635399, 0.5681284666061401, -2.299520969390869, 0.8163043260574341, 0.37213656306266785], [-0.6554720997810364, -0.08980382233858109, 0.6581709384918213, -1.908139705657959, -1.1694114208221436, 0.24032406508922577, -0.9499266743659973, -1.11122465133667, -1.5558743476867676, 0.14308558404445648, -0.21005582809448242, -1.9344303607940674, 0.9984537363052368, 1.0365016460418701, 0.5093869566917419, 0.7004920840263367, -1.22491455078125, 0.4289350211620331, 0.6956346035003662, -2.4560556411743164, -0.9962522983551025, -1.196137547492981, 1.2087340354919434, -1.4786653518676758, 0.4070618748664856, 0.4494889974594116, 0.2266152948141098, -0.1719575673341751, -0.3301604986190796, -1.8586280345916748, 2.465829372406006, -0.757016122341156, 0.6302108764648438, 0.3573436737060547, -0.8396836519241333, 0.3892483711242676, 1.5866466760635376, 0.4757485091686249, -0.439823180437088, 0.02805020846426487, -1.7968610525131226, 1.4328352212905884, -0.5795649886131287, 2.3641910552978516, -0.8380520343780518, 0.37585175037384033, -0.9058571457862854, -2.141201972961426, -0.34286990761756897, -0.6497970819473267, -0.1290217489004135, 0.640017032623291, 0.6384944319725037, 0.5393486618995667, -0.06966743618249893, -0.17024299502372742, -0.6562379598617554, -2.611375093460083, 0.49400436878204346, 1.0314304828643799, -1.6759166717529297, 0.9714468121528625, 0.49289679527282715, 1.1312100887298584], [-0.4709620475769043, 0.5978001356124878, -1.0313332080841064, -0.9816385507583618, -1.5278013944625854, -1.0247418880462646, 0.35823577642440796, -0.21337668597698212, 0.11088351160287857, 0.21028141677379608, -1.9514538049697876, -0.6055561900138855, -0.027604183182120323, 1.3163338899612427, -0.23152108490467072, 1.2430784702301025, -0.2820150554180145, 0.9172086119651794, -1.0554084777832031, -0.5763424634933472, -0.036022596061229706, -0.9720860719680786, -0.29638558626174927, -1.17999267578125, -1.7155911922454834, -1.988343596458435, -0.26756083965301514, -0.42370468378067017, -1.42186439037323, -0.10195222496986389, -1.290048360824585, 1.486297845840454, 0.6843361258506775, 1.6571955680847168, -0.03150557726621628, -1.6497347354888916, 0.974581241607666, 0.9127596616744995, 0.38419976830482483, 0.878544270992279, -0.41275298595428467, 1.8020530939102173, 0.623737096786499, -1.4816906452178955, -0.9290029406547546, -1.0673943758010864, 0.3324677646160126, 0.4536978006362915, -0.4423674941062927, 0.18922647833824158, -1.5574876070022583, -1.551753044128418, -0.41418370604515076, 1.3420608043670654, -0.9772000908851624, 0.14781253039836884, -0.6258673071861267, 1.4674561023712158, 1.6385438442230225, 1.3173964023590088, -0.1995517462491989, 0.276828408241272, -2.034496545791626, 1.9191017150878906], [0.12807022035121918, 0.32259389758110046, -0.17016282677650452, -1.5007065534591675, -0.3981788158416748, 0.16153252124786377, 0.5104122757911682, -0.7046485543251038, 0.9278682470321655, -0.34239843487739563, -1.4785252809524536, 0.3789371848106384, 0.26684945821762085, 0.6831399202346802, 0.3627435564994812, 1.6934691667556763, 0.022957615554332733, 0.11392183601856232, 1.4670164585113525, 0.6866713166236877, 0.6666111946105957, -0.29620474576950073, 0.903336763381958, -0.10056835412979126, -0.6281230449676514, -0.11431099474430084, 0.3685744106769562, 0.9694430828094482, 0.684783399105072, -0.05844637006521225, -0.717888355255127, 1.1622318029403687, -1.1195385456085205, 1.2880208492279053, 0.3918880820274353, -0.46895313262939453, 0.20991206169128418, 0.3114139437675476, -0.3525518774986267, 1.3527144193649292, -0.9526565670967102, -0.682068407535553, 0.2753017544746399, 0.18682456016540527, -0.08363643288612366, 0.015249254181981087, 1.7404025793075562, 0.43744996190071106, 0.5427535772323608, -0.6381171345710754, 1.2366708517074585, 0.5615071654319763, -0.8609710931777954, 0.4607880115509033, -1.0155893564224243, 0.5693792104721069, -1.2426868677139282, -1.298332691192627, -1.1982728242874146, 1.6563708782196045, -0.784974992275238, 0.3206901550292969, 1.1524635553359985, -0.6673831939697266], [0.1611073911190033, 2.1241157054901123, -1.2688337564468384, 0.21155817806720734, -1.5292339324951172, 0.12173143029212952, 0.8268672227859497, -0.7088460922241211, 1.0016112327575684, -1.1052095890045166, 1.0814929008483887, -0.8964836001396179, 1.2202521562576294, -0.9725859761238098, -1.336835503578186, -0.37523171305656433, -1.4363102912902832, 1.248822808265686, 0.43078985810279846, 0.024135647341609, -0.06324278563261032, -1.7489159107208252, -1.3273706436157227, -1.7821199893951416, 0.0998491570353508, -0.7594624161720276, 1.0196034908294678, -1.0937362909317017, 0.60886150598526, -1.770697832107544, -1.443621277809143, -0.30995938181877136, -2.474882125854492, -1.0537806749343872, -0.23772132396697998, -1.3675493001937866, 0.31385931372642517, -1.2491981983184814, 0.9325249791145325, -0.5338919758796692, -2.268214702606201, 1.8730149269104004, -0.3709372282028198, 0.6321427226066589, 1.241505742073059, -1.210218906402588, 1.7345952987670898, 0.22499851882457733, -0.27526241540908813, 0.26017630100250244, 1.2888479232788086, 0.4065636396408081, 0.31204384565353394, -0.861998438835144, 0.24576051533222198, 1.5760102272033691, -0.6491013765335083, 1.4825291633605957, 0.30641910433769226, 0.15478956699371338, 0.7948194146156311, -1.0118836164474487, 1.070613980293274, -0.04881402105093002], [0.23747670650482178, -0.005015698727220297, 0.4611380398273468, -1.5647085905075073, 0.3845718204975128, 1.6097581386566162, 0.35310566425323486, -0.42573195695877075, 0.4511716067790985, 0.37361428141593933, 0.33179813623428345, 0.9738828539848328, 0.42379093170166016, -0.34915295243263245, 0.37894338369369507, 1.31781005859375, 0.5503115653991699, 0.3152211010456085, -0.2891443073749542, 0.06410377472639084, 0.17523808777332306, 0.34399446845054626, 0.34619536995887756, 0.0036175900604575872, -0.8560488224029541, -0.042405568063259125, 0.989980161190033, -2.4133968353271484, 0.2754353880882263, -1.623454213142395, 1.708573341369629, -0.8368352651596069, 0.4779543876647949, -1.9630744457244873, -0.02611665427684784, 2.079688549041748, 0.7797860503196716, 1.2413649559020996, 0.6304813623428345, 0.952511727809906, 0.493935763835907, 1.3708754777908325, 0.9525538682937622, 1.059179425239563, -1.0122684240341187, -2.0536587238311768, -2.3404247760772705, -0.3041955530643463, 1.8123986721038818, 0.3458304703235626, 1.6732571125030518, 0.7200639247894287, 0.13391010463237762, 1.8650835752487183, 0.2773270010948181, 1.429652452468872, 0.47875547409057617, 0.15726399421691895, 0.40636536478996277, 0.25843244791030884, -2.4835331439971924, 1.8420182466506958, 0.8235591053962708, 1.4049921035766602], [-0.4535885453224182, 1.1997032165527344, 0.009734855964779854, 0.5201742649078369, 0.7956783175468445, -0.08335798233747482, 1.2084004878997803, -0.26794910430908203, -1.4864808320999146, -0.26339617371559143, -1.364149808883667, 0.10179190337657928, -0.3310096859931946, -0.2509169280529022, -0.37624120712280273, 0.5799084901809692, 1.858677864074707, 0.5272914171218872, -1.271115779876709, 2.7143983840942383, -0.17245718836784363, -1.797766089439392, 0.4635782539844513, -0.6971433758735657, 1.7413201332092285, 0.6463715434074402, -0.4836423397064209, 0.6645010113716125, -0.5586522817611694, -0.24932584166526794, 0.19528232514858246, -1.4644651412963867, 0.7693924903869629, 0.9081488251686096, 0.567724883556366, -0.14168889820575714, 1.2046226263046265, -0.32616156339645386, 0.4526071846485138, -0.6742749214172363, 0.25284871459007263, 0.34088149666786194, -0.8045766949653625, 0.7981179356575012, -0.39199864864349365, -1.9278950691223145, -0.5274043679237366, -0.7950176000595093, -2.1685192584991455, -0.8207976818084717, 1.3300644159317017, -0.9361377954483032, 0.8402613401412964, -1.2196307182312012, 0.15710672736167908, 1.0436590909957886, -0.3262813091278076, 1.813563585281372, -1.74407958984375, 0.6744220852851868, 1.3208707571029663, 0.4740241765975952, -0.47536501288414, 0.6651714444160461], [-1.295046091079712, 1.8932055234909058, -0.24063898622989655, 0.5232710242271423, 0.03894735500216484, -0.7510305643081665, 0.5152532458305359, 0.8310789465904236, -0.5145124197006226, 1.7900750637054443, 0.40042799711227417, -1.1127647161483765, 0.004746841732412577, 1.5326008796691895, -0.6548555493354797, 0.7415105104446411, -1.1683560609817505, 0.9983747005462646, -0.6252925395965576, -0.15082304179668427, 1.2347584962844849, -1.183842420578003, 1.2926756143569946, -0.36417052149772644, 0.18691003322601318, -0.011422224342823029, -1.9103412628173828, -0.6525245308876038, 1.1304152011871338, -0.06314120441675186, -0.3000181019306183, -0.9431975483894348, 1.2103779315948486, 0.05681406706571579, 1.2963141202926636, -0.40841832756996155, 0.7879193425178528, -0.8172641396522522, 0.23317956924438477, 0.04097771272063255, 0.3497871160507202, 0.9297817945480347, 0.7749398946762085, 0.5118867754936218, 0.9045547842979431, 1.217383861541748, 1.4016422033309937, 0.07508086413145065, 0.5228492617607117, 0.9293471574783325, -1.0296621322631836, 2.5227608680725098, -1.3707842826843262, 0.32360491156578064, 0.19366039335727692, -2.127035617828369, 0.27997180819511414, -0.45368653535842896, 0.8738378286361694, 0.7014725804328918, 0.20545729994773865, -1.1443425416946411, 0.8731318116188049, 1.3024920225143433], [-0.04985848814249039, -0.3397867679595947, -0.5021437406539917, -0.634509265422821, 0.2342051863670349, -0.9454934000968933, 0.8486713171005249, 0.03334851935505867, -1.7642465829849243, 2.122955083847046, -0.6715819835662842, -0.05402014032006264, 0.9936224818229675, 0.4604688286781311, 1.4175639152526855, -0.416683554649353, -0.8231423497200012, 0.5132997632026672, -0.8153338432312012, 0.888787567615509, 0.6179490089416504, 0.33207643032073975, -1.0186889171600342, 1.6905882358551025, -0.6680447459220886, -2.6855084896087646, -0.9425643682479858, 0.38041412830352783, -0.5395040512084961, 1.059373140335083, -0.3635529577732086, 1.7105214595794678, 0.22796127200126648, 1.7925442457199097, -0.12295873463153839, -0.26026564836502075, 0.8218864798545837, 0.1967342495918274, -1.1197905540466309, -2.111643075942993, 0.5594379305839539, -0.38929665088653564, -0.4631217122077942, -0.6581801176071167, 1.4010021686553955, -0.3592468202114105, 0.13948474824428558, -1.0780096054077148, -0.5183508992195129, 0.13294430077075958, 0.3120139241218567, 0.05743194371461868, 0.5676029324531555, 0.21446533501148224, 0.4083713889122009, -0.29488319158554077, -1.9996353387832642, -0.2418498992919922, -0.2759462893009186, 0.1159532219171524, 0.6901050209999084, 2.174058437347412, -0.8725799322128296, 1.4370614290237427], [-0.3111710548400879, 1.1451948881149292, 0.12768416106700897, -0.6812407970428467, 0.44698676466941833, 1.1356805562973022, 0.39024072885513306, 0.31308484077453613, -0.8522323966026306, 1.1184698343276978, 0.8033875823020935, 0.6303942799568176, 0.4164706766605377, -0.5584692358970642, -0.21772833168506622, 0.18522785604000092, 0.3134499490261078, 0.683648407459259, 0.4642549455165863, 1.775234341621399, -1.0978689193725586, 0.41219550371170044, -1.7330013513565063, 1.420595407485962, 0.007322276011109352, -1.7232270240783691, -0.462681382894516, 0.25613027811050415, -0.08130688220262527, 0.2640365958213806, -0.23464007675647736, -0.004259598441421986, 1.2996381521224976, 1.1715728044509888, -1.436404824256897, 0.05624557286500931, -0.7108107805252075, 1.814438819885254, -0.3023330867290497, 0.3211785852909088, 0.16748876869678497, 0.8002296686172485, 0.5429050922393799, 1.4218589067459106, -1.3005366325378418, 0.2158997803926468, 0.13130086660385132, 0.15276822447776794, 0.5787218809127808, 0.004468834493309259, 1.2896960973739624, -0.001743673114106059, -1.372506856918335, 0.5783933401107788, -1.4410344362258911, 1.7190992832183838, -0.9158951640129089, 0.6062275171279907, 0.003835233161225915, 0.7450447678565979, 0.4790533185005188, -2.473362684249878, -0.5092004537582397, 0.09199204295873642], [-0.7782701849937439, -0.33278417587280273, 1.051123023033142, -0.2856437563896179, -1.7315640449523926, 1.1183456182479858, 0.6871495842933655, 0.4291432201862335, 3.5900306701660156, -0.6106428503990173, -0.13752596080303192, -0.944937527179718, 0.016109324991703033, 0.09883265197277069, -0.488473504781723, 0.772642195224762, 0.7137534022331238, -0.15752457082271576, 1.3376141786575317, 1.3928810358047485, -0.2775849401950836, 2.1718716621398926, 0.14104127883911133, 0.5905545949935913, 2.089261293411255, -0.09918155521154404, 0.14469099044799805, -1.4329936504364014, -0.7906222343444824, -0.7462394833564758, 0.03976673260331154, -1.61370849609375, -0.2937944829463959, 1.0699632167816162, -1.9607378244400024, 0.8822194337844849, -0.5543262362480164, 1.5928329229354858, 0.27935904264450073, 1.0976336002349854, 0.7416213750839233, -0.18748007714748383, -0.17611171305179596, -0.9817979335784912, 0.5549483895301819, 0.43229931592941284, 0.33072054386138916, 0.26448726654052734, -0.2622755765914917, 0.4490705132484436, -1.6995002031326294, 0.17047366499900818, -0.49611344933509827, -0.6230339407920837, 0.794367790222168, 0.41318008303642273, -0.037108320742845535, -0.1659335047006607, -1.143574595451355, 0.03643923997879028, 1.162160038948059, 2.02061128616333, -0.7480385899543762, 1.536799669265747], [1.653041124343872, 1.002856969833374, -1.669788122177124, 0.011371695436537266, 1.3447344303131104, -0.9789256453514099, -0.2464839220046997, 1.6452641487121582, -0.0416434146463871, 0.2657538056373596, 0.5588991641998291, -0.570357620716095, -0.5113478302955627, -0.8179057240486145, 1.3332056999206543, 0.3715178072452545, -1.1051815748214722, 0.4856111407279968, -0.47811031341552734, 0.15331238508224487, 0.3097774386405945, 1.6526778936386108, -0.2582016885280609, -0.49003317952156067, 1.27527916431427, -1.890073299407959, -0.02483942359685898, -0.10143726319074631, 0.26382410526275635, -1.841972827911377, 0.018231991678476334, -0.05470530688762665, -0.2799774408340454, 1.421744465827942, -0.280826598405838, -1.439787745475769, 2.0539257526397705, -0.40919074416160583, 0.8328791260719299, -0.2297441065311432, 0.7910094261169434, 1.8843228816986084, -0.08880068361759186, 0.9460473656654358, 1.3078134059906006, 0.26232191920280457, 0.5482475757598877, 0.5136613249778748, 0.513173520565033, 0.06658611446619034, 1.1059989929199219, 2.405214309692383, 0.8265049457550049, 1.040769100189209, 2.835505962371826, -1.2414883375167847, 0.7354689836502075, 0.79448401927948, 0.942389726638794, 0.11404592543840408, 0.5364802479743958, -0.46116912364959717, -0.5899806618690491, 0.2927228510379791], [-0.02635030820965767, -1.108944296836853, 0.36135104298591614, 0.05762692540884018, -0.5262242555618286, 1.8194165229797363, 1.4713094234466553, -0.1030813455581665, 1.2322607040405273, 0.38809332251548767, 0.06678666919469833, 0.1320379227399826, -0.844478189945221, 2.0030150413513184, -1.088465690612793, -0.7575864195823669, 1.8129185438156128, 0.19872769713401794, 2.5238800048828125, -0.07378602772951126, -1.0660068988800049, 1.0241152048110962, 0.4469335675239563, 1.055650234222412, -1.6522538661956787, -1.9294382333755493, -0.7220494151115417, 0.6052505970001221, 1.1754117012023926, -1.3343260288238525, 0.4449116289615631, 0.6095583438873291, -0.7003529667854309, -0.2802432179450989, -1.1640268564224243, 1.2082937955856323, -0.1980654001235962, -1.790827751159668, -0.038007356226444244, -0.269377201795578, 0.7181842923164368, -0.16769807040691376, 0.6344953179359436, 1.0409975051879883, 0.5258249044418335, -1.8922145366668701, -1.2062468528747559, 1.1174100637435913, 0.3129449188709259, -1.3669332265853882, -0.782370388507843, 0.22006243467330933, -0.46668344736099243, 0.4310779273509979, -1.5656957626342773, -0.7467498779296875, 0.1916830688714981, 0.4788825213909149, 0.5238335728645325, 0.9562500715255737, 0.8901346921920776, 1.8931987285614014, 0.46744605898857117, 0.6670022010803223], [0.3867042064666748, 0.379511296749115, -0.853787899017334, -0.5575050115585327, 0.6548422574996948, -0.2088479846715927, -0.3890317678451538, 0.945003092288971, -0.8323227763175964, 1.0508484840393066, -0.1747770458459854, 1.0359119176864624, 0.48692652583122253, -0.21531586349010468, 1.288473129272461, -1.0660847425460815, 0.287274569272995, -0.3030601143836975, -0.7299318909645081, -0.6848241686820984, 1.056553602218628, 0.2891779839992523, 2.4783804416656494, 1.1258662939071655, 0.04018807411193848, 1.0774672031402588, -0.6474048495292664, -1.6103332042694092, 0.7966737151145935, 0.2325059026479721, 1.1612844467163086, -0.7838597893714905, -0.4737841486930847, -1.814957618713379, -0.3216233551502228, -0.3654772937297821, -1.1563806533813477, 0.24784906208515167, 0.6581113934516907, -0.6840013265609741, -0.9032312035560608, -0.051203653216362, -0.31117501854896545, 0.2688944339752197, -0.42566967010498047, 1.8804467916488647, 0.10150003433227539, -1.4245314598083496, 0.04606650397181511, -0.06478864699602127, -0.5450990796089172, 0.12592381238937378, 0.21135267615318298, 0.09448881447315216, 0.7834474444389343, 0.09718126803636551, 0.4601494073867798, 0.630899965763092, 0.027560634538531303, 1.0401809215545654, 0.2604954242706299, -1.4710562229156494, 0.04595768451690674, -0.16531364619731903], [0.6759099960327148, -0.47699904441833496, 0.9118061661720276, -0.26529550552368164, -2.1528916358947754, 1.1409279108047485, -1.3964911699295044, 1.169792652130127, -0.8605605959892273, -0.9914589524269104, 0.19644196331501007, -1.7489571571350098, 0.4096956253051758, -0.42991045117378235, -0.3563332259654999, 1.6604695320129395, 1.2270864248275757, 0.09057486057281494, -1.107778549194336, -0.1085454672574997, 0.7410527467727661, 0.6370599865913391, -0.969398558139801, -1.1438484191894531, -1.381508469581604, -1.0064204931259155, 1.473454475402832, 0.018630463629961014, -1.1909934282302856, 1.3136252164840698, 0.21508972346782684, 0.6593505144119263, -0.01996336132287979, -0.343033105134964, -0.33828774094581604, 0.04639139771461487, -0.16661007702350616, -1.1261239051818848, 0.3144543468952179, 0.02341889962553978, 0.7647983431816101, 0.4487936198711395, -0.5950420498847961, -0.7441568970680237, 0.28385430574417114, 0.2501778304576874, 0.4610731601715088, 0.28381457924842834, 1.2522023916244507, -1.323307752609253, 1.023730993270874, -1.6200553178787231, -1.9005626440048218, -0.7456006407737732, -1.0562710762023926, 2.1779706478118896, 0.48092547059059143, -0.6286779046058655, -0.3809395134449005, 1.5129709243774414, -0.4884331226348877, 0.9400769472122192, -0.07324475049972534, -0.4476456344127655], [1.845084547996521, -0.85959792137146, 0.07800786942243576, -0.2443341463804245, -0.058765310794115067, -1.2046265602111816, -1.134036898612976, -0.23525559902191162, 1.6349365711212158, -1.1299787759780884, -0.32779359817504883, 0.881095826625824, -0.49745890498161316, 0.3149285912513733, 1.876625657081604, 0.7616404294967651, -0.3628411293029785, -0.5241549611091614, 1.437788486480713, -0.04785526916384697, 1.5853339433670044, 0.30105313658714294, -0.031476981937885284, 0.34374117851257324, 0.6418388485908508, 0.8940712809562683, 1.0948747396469116, -0.4923792779445648, 0.8220909833908081, -0.5838518738746643, -0.9529324173927307, 1.6701970100402832, 1.2509827613830566, -3.15564227104187, 0.6039619445800781, -0.407762736082077, -0.4285971224308014, 0.12584832310676575, -0.2410716563463211, -0.25930511951446533, 0.15494032204151154, 1.0603954792022705, -1.0040969848632812, 0.7849188446998596, 0.8848636150360107, -1.242014765739441, 0.19483914971351624, -2.045457124710083, -0.41857174038887024, 1.1221168041229248, 0.3963688015937805, 0.44864484667778015, -1.410000205039978, 1.9285107851028442, 0.682235836982727, 0.9729952216148376, -0.27529796957969666, -0.8249610066413879, -0.6040798425674438, -0.9894387125968933, -1.0567200183868408, -0.825734555721283, 0.023531120270490646, -0.37151822447776794], [-0.8308696746826172, 0.13589514791965485, 0.358348548412323, 1.3210960626602173, 2.0935187339782715, 1.1637663841247559, -1.059468150138855, -0.4043562412261963, -0.7662186026573181, -0.8286113739013672, 1.3536256551742554, -0.9656608700752258, -0.10813233256340027, -0.5653575658798218, -0.09793496876955032, 0.7731520533561707, -0.09892192482948303, -0.27731582522392273, 2.0504369735717773, 0.013182776048779488, -0.4215596914291382, -0.18163026869297028, -1.6632314920425415, -0.28574544191360474, 1.515972375869751, 0.6876099109649658, -0.3191959261894226, -0.982746422290802, 1.6392834186553955, -0.71153724193573, 0.37398630380630493, -0.5657638907432556, -0.7592275738716125, -3.0141468048095703, -0.9199926853179932, -0.5191985368728638, -1.212059497833252, 0.703216552734375, -0.1785970777273178, -0.6546770930290222, -1.732954978942871, 1.835906982421875, -1.3154503107070923, 0.47385069727897644, -1.1433765888214111, -0.4496743977069855, -0.4823632538318634, 0.9536768198013306, -0.18076397478580475, 1.2653988599777222, 1.4713494777679443, 0.22604933381080627, 0.2145547717809677, 0.08556697517633438, -0.23485073447227478, -1.4659346342086792, 1.4817994832992554, 0.30440181493759155, 2.047300100326538, -0.06636656820774078, 0.11177460104227066, 0.35840940475463867, -0.5573891401290894, -1.089264154434204], [1.169308066368103, 0.6148438453674316, -1.3402937650680542, -0.021815750747919083, -0.20700211822986603, 1.0302915573120117, 0.08347082138061523, 1.0751157999038696, -0.8945608735084534, -0.12135578691959381, -1.2486578226089478, -0.6585355997085571, -0.010759689845144749, -0.0743248388171196, 0.05407803878188133, -0.7507277131080627, -2.3469038009643555, -0.7242279648780823, -1.0271371603012085, -0.08880847692489624, -0.7389721274375916, -0.3292480409145355, 1.8301254510879517, -0.34852084517478943, 0.9118797183036804, -0.4004286825656891, 0.6326497793197632, -1.0655039548873901, 0.5995212197303772, 0.6151353120803833, 0.7837092876434326, -0.8273864388465881, 0.3187367618083954, 1.5095655918121338, 0.0032420570496469736, 0.6679667234420776, -1.6458390951156616, 0.3871672451496124, 0.1356678307056427, -0.13031169772148132, -0.23738066852092743, 0.008518041111528873, 0.3645666241645813, -1.3688963651657104, 1.5746262073516846, -0.2784591317176819, -0.783211350440979, 0.7537323832511902, 1.2721363306045532, -0.4414186477661133, 1.6797499656677246, 3.254385471343994, 0.9209663271903992, 1.5449986457824707, -0.5837214589118958, -1.660829782485962, -0.6506636142730713, -1.4737468957901, 2.228670358657837, 0.5217923521995544, -0.18246303498744965, -0.19391627609729767, 0.18313688039779663, 0.14611229300498962], [-0.7090622186660767, 0.5092360973358154, 0.7510586977005005, 1.2512766122817993, 0.7961310744285583, 1.7833929061889648, -0.23347875475883484, 0.0010392919648438692, 0.6370965242385864, 0.5094374418258667, -1.3051635026931763, 1.03646719455719, -0.21297483146190643, 0.06943345069885254, 0.2813616394996643, -1.3612468242645264, -0.23404152691364288, 0.7531501054763794, 0.850538432598114, -0.2624572813510895, -0.15763014554977417, -2.883972644805908, 0.8251025676727295, -0.18298782408237457, 1.1177787780761719, -1.2837027311325073, 0.8797317743301392, 0.3664907217025757, 1.6933631896972656, 0.825950562953949, -0.7644577622413635, 1.1796016693115234, 0.5526677370071411, 0.4838196635246277, 0.045818544924259186, 1.5612114667892456, 0.10641034692525864, -0.5072967410087585, 2.080091953277588, -0.48196128010749817, 0.715130627155304, 0.5490211844444275, -1.2639169692993164, 1.467738151550293, 2.0499041080474854, -0.03147700056433678, -1.9147663116455078, -0.6013829708099365, -0.6699097752571106, -1.9912610054016113, 0.8965283632278442, -0.9671737551689148, -0.17785663902759552, 0.10522786527872086, -0.2769181728363037, 0.5250465273857117, 0.23796668648719788, -0.4944632947444916, -0.0846322700381279, -0.5369871854782104, 1.3856958150863647, 0.0577336810529232, -0.8516349792480469, 1.162892460823059], [0.4718904197216034, -0.318121075630188, -0.12614735960960388, -0.9866676926612854, 0.5615211129188538, 0.47980767488479614, 0.6311533451080322, -1.1897090673446655, -0.7100014686584473, 0.0005878410302102566, 1.253198266029358, 1.197758674621582, 0.023774152621626854, 1.6790090799331665, 0.6936565637588501, 1.0273199081420898, 0.7122865319252014, 0.15117962658405304, -0.2738364338874817, -3.2232842445373535, 0.20847128331661224, 0.21407529711723328, -0.6044088006019592, 0.47226813435554504, 0.2454894334077835, -1.5390783548355103, -0.48638010025024414, 0.9573605060577393, 0.8523033261299133, 0.05514013022184372, -0.7723191976547241, 0.7786441445350647, 0.8444156050682068, -1.7446502447128296, 0.46847888827323914, -0.9943492412567139, -0.3754800260066986, -0.691814661026001, -0.7011802792549133, 0.8068234920501709, -0.4462433159351349, 1.2926278114318848, 0.5049639940261841, -0.8306816220283508, -0.9240027070045471, 0.6176628470420837, 0.5901171565055847, -0.007728918921202421, 1.6991912126541138, 0.45875921845436096, 0.39806684851646423, -0.11324188858270645, -0.8425338864326477, 0.8472471833229065, -1.8527740240097046, 0.7994763851165771, 1.2159088850021362, -1.1153923273086548, -0.12542536854743958, 0.735100507736206, 0.1859433948993683, 0.7484889030456543, -1.1322368383407593, -1.2897330522537231], [-0.6568038463592529, 0.30772337317466736, -2.4047868251800537, -0.761288583278656, 0.9478424191474915, -0.9488045573234558, 1.0110472440719604, -0.47102993726730347, 0.027690263465046883, -2.422811985015869, -1.4370986223220825, -0.07217719405889511, -0.6559147238731384, 0.18521329760551453, -1.0208944082260132, -0.8502160310745239, -0.5376945734024048, 1.2060340642929077, 0.5203067064285278, -0.11877230554819107, -1.5306086540222168, 1.172379732131958, -0.9263961911201477, -0.9023116827011108, 2.0495007038116455, -1.168520450592041, -0.2596193552017212, 2.0122878551483154, 0.9463542103767395, -1.4357556104660034, 0.7492859959602356, 1.4372532367706299, -1.5472924709320068, -1.3555837869644165, 0.998354971408844, -0.530075192451477, 0.038409218192100525, -0.6115124821662903, -0.6588839292526245, -0.4232359230518341, 0.4289652705192566, -0.6643564105033875, 0.5767146348953247, -0.5403609275817871, -1.0787872076034546, 1.1339292526245117, 0.136018767952919, -0.13873982429504395, 0.038041554391384125, -1.9820457696914673, 1.7050672769546509, -0.27561116218566895, -0.24823641777038574, 0.6791414618492126, 0.46266457438468933, -1.0772017240524292, 1.2038174867630005, -0.848903238773346, -0.5945563316345215, 2.3461968898773193, -1.1503814458847046, -0.9998682141304016, 2.064984083175659, -1.3813122510910034], [0.7404112219810486, -2.545944929122925, 0.3595495820045471, 0.6191433668136597, 0.2743661105632782, 0.2763092815876007, 1.4603900909423828, 0.14093557000160217, 0.5664117932319641, 1.3663995265960693, 1.7604854106903076, -0.5672323107719421, 0.4563908576965332, 1.379517912864685, -1.5035052299499512, 0.335741251707077, 1.041747808456421, 1.411159634590149, 0.30935394763946533, -0.012364641763269901, -0.5494691729545593, -0.8905661106109619, -1.0365018844604492, -1.1502212285995483, 2.010488986968994, -0.17710967361927032, -0.4348985552787781, -0.31060102581977844, -0.4282727837562561, -0.3490400016307831, -0.8238662481307983, 0.5887812376022339, 2.4232261180877686, -2.0227913856506348, 0.49120432138442993, -0.22899897396564484, -0.5176899433135986, -1.1087614297866821, -1.8751167058944702, 0.39627963304519653, -2.2180349826812744, 0.9443244338035583, 1.131871223449707, 0.7326732873916626, 0.9992559552192688, -0.8914133310317993, 0.6900714635848999, -1.702170491218567, 0.7187659740447998, 0.00949755311012268, -2.0415115356445312, 1.5840009450912476, 0.16303296387195587, 0.9394199252128601, -0.32048413157463074, -0.8509625792503357, -0.21696561574935913, 0.24564680457115173, 0.587932825088501, 0.14286135137081146, -0.4345884621143341, 1.944827675819397, -0.3272647261619568, -1.233094334602356], [0.4813685417175293, -1.9232685565948486, -0.6842502951622009, 0.26401427388191223, 2.3056437969207764, -0.1571713387966156, 0.22562424838542938, 0.3302823305130005, -1.3842412233352661, 0.8713155388832092, 1.8883121013641357, 0.04872269183397293, 0.05507984012365341, -1.264082431793213, -2.372562885284424, 0.3294784128665924, -0.2510828375816345, 0.05657158046960831, 1.0048412084579468, -0.8241196274757385, 1.3098965883255005, -1.8747737407684326, 0.10490481555461884, -0.05709078907966614, -0.6645815372467041, 0.23919005692005157, -2.6203982830047607, -1.9046978950500488, -0.9500109553337097, -0.16701866686344147, 0.7938904166221619, -1.08154296875, -0.9584404230117798, -1.2860214710235596, 0.9955344796180725, -1.3175711631774902, -0.8380767703056335, -0.3535875976085663, -1.094092845916748, -1.842232346534729, 1.3009181022644043, -0.1137160137295723, -1.119802474975586, -1.0910735130310059, 0.3142918348312378, 0.6520286202430725, -1.2976830005645752, -1.7442578077316284, -0.28824806213378906, 0.4840560853481293, -1.4151824712753296, 1.565688967704773, 0.1587701439857483, 0.19186052680015564, -1.0459349155426025, 0.1617107391357422, -1.30557119846344, -1.6254138946533203, 0.23821808397769928, -0.35641807317733765, 1.0217119455337524, -0.39484903216362, 0.13056354224681854, 2.285586357116699], [-0.8765462636947632, -0.3394632935523987, 1.3437221050262451, -0.758764922618866, -0.16207611560821533, -0.2945480942726135, 0.29383987188339233, -0.3761399984359741, -1.0927796363830566, -0.2621121108531952, -1.3419067859649658, 1.3923213481903076, 0.2122117131948471, -0.7033413052558899, 2.081451892852783, 0.0753960907459259, 0.7032286524772644, 0.1732373684644699, -0.9080953598022461, 1.4576302766799927, -2.5790631771087646, 0.1509355902671814, 1.1399767398834229, 0.5237396955490112, 1.5707868337631226, -0.01550030056387186, 0.7810048460960388, 2.185462713241577, 0.16060766577720642, -0.9177528619766235, 0.4722859263420105, -0.5447880029678345, -0.6045514345169067, -1.5470670461654663, -0.5532204508781433, 1.130515456199646, -0.948350727558136, -1.2804152965545654, -1.0938479900360107, 0.13887213170528412, 1.0623329877853394, 0.346637487411499, 0.5452305674552917, -1.0078693628311157, -1.366852879524231, 0.6592673063278198, -0.7081875205039978, -0.21934740245342255, 0.8814078569412231, 2.14949369430542, -0.8405584096908569, 0.7560888528823853, 1.2730638980865479, -1.8443289995193481, -1.8116494417190552, 0.10458044707775116, -1.3195745944976807, -2.832047939300537, -1.2131749391555786, 0.9582298398017883, -1.0893419981002808, -0.5035399198532104, -0.16573606431484222, -0.2884785532951355], [-0.18667037785053253, -2.277829647064209, -0.3999919295310974, 1.0847480297088623, -0.0339539498090744, -1.7469940185546875, 0.25648847222328186, 0.4178607761859894, 1.1678987741470337, -0.42272475361824036, 2.144930601119995, 1.34572172164917, -1.8660255670547485, -0.8330346941947937, -0.5049885511398315, -1.9831054210662842, 0.008284988813102245, 0.4016527831554413, -1.5200910568237305, -0.5782527923583984, -2.1514618396759033, -0.11967014521360397, -0.400704562664032, 1.3551318645477295, 1.4911537170410156, -0.19564491510391235, 0.27804288268089294, 0.9963805079460144, -1.4308511018753052, -0.8638611435890198, -0.32553398609161377, 0.45255574584007263, 0.6773334741592407, -1.5567327737808228, -0.8296738266944885, -0.6836979389190674, 0.21223996579647064, 1.272652268409729, 1.0600957870483398, 1.0922901630401611, -1.784803032875061, -0.8051708936691284, 0.8361167311668396, 1.100799798965454, 0.04107743501663208, 0.028851885348558426, 0.46419134736061096, -2.0143203735351562, 0.13631489872932434, -0.47971412539482117, 0.7651938199996948, 1.211514949798584, -1.124082326889038, 0.01964648999273777, 0.0265194159001112, 0.37720564007759094, 0.04424121975898743, 0.4158869683742523, 0.09559280425310135, 0.014690320938825607, -0.6016683578491211, 1.8229014873504639, -0.16997915506362915, -0.85237717628479], [0.6854720115661621, 2.8760554790496826, -0.918477475643158, 2.001890182495117, 0.5432737469673157, -0.4184032082557678, -0.06702034920454025, -1.2565778493881226, 0.19483883678913116, 0.08345307409763336, -0.1279069185256958, -0.26778995990753174, 0.3095163404941559, -0.015886131674051285, -0.4769953191280365, 1.3525471687316895, -0.6153107285499573, -1.322543978691101, -0.6762918829917908, 0.36281299591064453, -0.5201438069343567, 0.504887580871582, -0.22120659053325653, -0.7036771178245544, -1.5584675073623657, -0.515803873538971, -1.0067119598388672, -0.16772441565990448, 1.5266354084014893, 1.3049291372299194, 1.3386540412902832, -0.17774377763271332, 0.9640223383903503, -0.7584581971168518, 0.8240101933479309, -0.7438074946403503, 0.3196814954280853, -2.324643850326538, 1.372681975364685, 0.46577417850494385, -0.8819044232368469, -0.06530354171991348, 2.053069829940796, -1.193345308303833, 1.4757462739944458, 0.04102098196744919, 0.7734062671661377, 0.7211618423461914, -0.4149978458881378, 0.3485647439956665, 0.4921787977218628, 0.5125176310539246, -0.272362619638443, 0.9981760382652283, -0.18034566938877106, 0.5654672384262085, 1.0633286237716675, -0.5544968247413635, -0.41353839635849, -0.23349781334400177, -0.5465195178985596, 0.9791805744171143, -2.2418484687805176, -0.08700361847877502], [0.634696364402771, -0.18364903330802917, -0.2718906104564667, -0.696029007434845, -1.001590371131897, -2.5547471046447754, -0.5242056250572205, -1.511526107788086, 0.8814967274665833, -1.0431177616119385, -0.2679501175880432, -0.5483548641204834, 1.3558672666549683, 0.13531707227230072, 2.4248173236846924, -0.9855796098709106, -0.1285635083913803, 0.6188541054725647, 0.3021063208580017, -1.5408369302749634, 1.3985179662704468, 0.2887454032897949, -0.4191294014453888, -0.41687458753585815, 0.39396166801452637, -0.11507785320281982, -1.117964744567871, 1.4611543416976929, 0.3231717646121979, 0.7472072839736938, -0.3660566806793213, 0.823729395866394, 0.29395756125450134, -0.5477237105369568, 1.7702709436416626, -1.239980936050415, 2.2040717601776123, -0.0748993381857872, 1.7158175706863403, 0.29860904812812805, 0.20785684883594513, -0.6444187760353088, -1.0616108179092407, 0.9701571464538574, -0.5296282172203064, -0.4492112994194031, -0.91082763671875, 0.14664864540100098, 0.41609349846839905, -0.039742495864629745, -0.3150504231452942, 2.0462701320648193, -0.12373902648687363, 0.5655903816223145, 1.057984709739685, -0.345535010099411, 0.5887008905410767, -1.0137332677841187, -2.111144542694092, -1.8253589868545532, -0.7514343857765198, 0.3745979368686676, 0.14028745889663696, -0.09628196060657501], [0.8748962879180908, 0.4328111410140991, 1.5605740547180176, 0.12720650434494019, -0.4585813283920288, 0.39260897040367126, 0.644835889339447, -1.7176843881607056, -1.2425118684768677, 0.0106212068349123, 0.0824376568198204, -2.461104154586792, 1.3189927339553833, 0.7909788489341736, 0.42366087436676025, -0.9337347149848938, 3.105520486831665, -0.8413130044937134, 0.2844262421131134, -1.3301113843917847, -1.397282600402832, -0.7135160565376282, -0.6569783687591553, -0.7180788516998291, -0.2027883380651474, -0.7374377250671387, -0.3874538838863373, -0.5865892171859741, 0.3396452069282532, -0.18965837359428406, -0.25007182359695435, 0.8213512301445007, 0.2541833221912384, -1.8540312051773071, 1.5116287469863892, 2.444944381713867, -0.38003650307655334, 0.19917413592338562, -0.5779613852500916, -0.5903445482254028, 1.88179612159729, 1.0921940803527832, 1.5538246631622314, -0.7107464075088501, -0.28057214617729187, -1.0679664611816406, 1.5100598335266113, 2.075639247894287, 0.5387250185012817, -0.45028355717658997, -0.2407115250825882, 0.0015685209073126316, -0.6562775373458862, -1.0008174180984497, -1.143696665763855, 1.3164870738983154, -0.802632749080658, -0.7174712419509888, -0.37498706579208374, -0.2464590221643448, -0.8497275114059448, -1.3946144580841064, -0.9185755252838135, 0.697822093963623], [-2.3384454250335693, 1.2844550609588623, -1.0103963613510132, 2.285775661468506, -0.33044490218162537, 1.871932029724121, -2.4327988624572754, -0.15470850467681885, -0.5429169535636902, -0.7510964870452881, -1.6367084980010986, -1.7727770805358887, 1.6123058795928955, -1.8808796405792236, 0.7089002132415771, -0.30391690135002136, 0.1780991405248642, -0.083333820104599, -0.5781594514846802, -0.40143513679504395, -1.3055143356323242, 0.016774136573076248, -1.4294496774673462, 0.5506229996681213, 1.1005678176879883, -1.3325945138931274, 0.5975087285041809, -0.17867375910282135, 0.664000391960144, 0.6204298138618469, -0.8280532956123352, 0.9269681572914124, -0.9552202820777893, 0.5305190682411194, 0.8684766888618469, -0.17428426444530487, -0.2861725389957428, -0.9099602699279785, 0.19198942184448242, 1.1864410638809204, -0.07969103753566742, -0.778972327709198, 0.6905112266540527, -0.9725039005279541, -0.9860866665840149, -1.1002275943756104, -1.0941603183746338, 1.7398563623428345, -0.5955349206924438, -0.5581694841384888, -0.6516013741493225, -0.3746126890182495, 1.6312255859375, 0.2655600607395172, -1.0197651386260986, -0.23995865881443024, -1.2375636100769043, 0.12198218703269958, 0.9355037212371826, -0.7175867557525635, 0.5042117834091187, -2.963599681854248, 1.430098056793213, 0.8524153232574463], [-0.3978980779647827, 0.9568753242492676, -0.07192196696996689, -0.20032039284706116, 0.23230324685573578, -0.8010202646255493, 0.9022198915481567, 0.12409186363220215, -0.3964686393737793, 0.7066900730133057, -0.5372374653816223, -0.7650569677352905, -0.15344202518463135, -0.1898971050977707, 0.39177173376083374, -1.4837168455123901, -0.7638886570930481, -0.7752261757850647, -0.6498770713806152, -0.9362421035766602, -0.538335382938385, -1.5635892152786255, -0.5632586479187012, 0.6582701802253723, 0.9072538614273071, 0.6641719937324524, -0.7455567717552185, 0.10570719093084335, -0.6929863691329956, -0.4238370954990387, 1.3904683589935303, 1.3194304704666138, -0.42703959345817566, 1.3440126180648804, -1.4218366146087646, 1.6862343549728394, 0.11379502713680267, -0.6847774386405945, -0.038149796426296234, 0.9791609644889832, -0.5432764887809753, -0.09333807975053787, -0.9657112956047058, 3.1039905548095703, -0.8653138279914856, 2.0267207622528076, 0.32600390911102295, -1.0505845546722412, -1.805675983428955, -1.4346890449523926, 0.9219834804534912, -0.5279873013496399, -0.19029809534549713, 1.7953585386276245, 0.6311150193214417, 0.5501692891120911, -0.014128029346466064, -1.1545871496200562, 0.1566210389137268, 0.29310357570648193, -0.29062408208847046, 1.176574468612671, 0.9066005349159241, -3.068207025527954], [-0.5823999643325806, 0.08707725256681442, -0.3235219419002533, -1.2223868370056152, -0.46389564871788025, -0.6484278440475464, -0.16403266787528992, -0.8174306154251099, 0.5759343504905701, 0.6148720383644104, -0.0344027541577816, 3.1424102783203125, 0.3302760124206543, -0.24576298892498016, -0.16467030346393585, -0.5515239238739014, 1.3760958909988403, 0.22000929713249207, 0.2942168116569519, 0.6276971697807312, 0.1083197221159935, 0.4051584005355835, 1.505475401878357, 1.380956768989563, 0.17231330275535583, 0.06854161620140076, 0.36022794246673584, 1.1117939949035645, 1.7446473836898804, -0.30457425117492676, -0.036739788949489594, 2.11557674407959, 3.13895320892334, -0.747360348701477, 0.5821497440338135, -0.2869632840156555, -0.14311183989048004, -0.21688413619995117, 0.7184097766876221, 1.4115772247314453, -1.4332526922225952, 0.12871208786964417, -0.1873459815979004, 0.5125148892402649, -0.5038459897041321, -0.06325530260801315, -1.9578248262405396, -0.71589595079422, -0.2955566346645355, -0.2982666492462158, -0.5137678384780884, 1.167012095451355, -0.5201072096824646, -0.6265203952789307, 1.6126408576965332, -0.41862139105796814, -0.8433710336685181, -0.8308367133140564, -0.7073163390159607, -0.8047978281974792, 0.6283586025238037, -0.32647305727005005, 0.8940820693969727, 2.3492531776428223], [0.5712804198265076, -0.37411168217658997, 1.278938889503479, -0.220720112323761, 1.0641179084777832, -0.31451988220214844, -1.9886358976364136, 0.8987466096878052, 0.603004515171051, 0.9012967348098755, -0.639773428440094, 1.3228715658187866, -0.15271250903606415, -0.2430814504623413, -0.04609740152955055, 0.21036630868911743, -2.4831759929656982, -0.032007504254579544, -0.8702691197395325, -0.40398135781288147, -0.6363216042518616, -0.05876810848712921, 0.07481124997138977, -0.06794191896915436, -1.927740216255188, 1.4777418375015259, -0.17787382006645203, 0.7596766948699951, -0.17533478140830994, -1.687633752822876, 0.5432701110839844, -1.5741000175476074, 0.6509013175964355, 0.1813051849603653, 0.43752211332321167, 0.6369050741195679, 0.4405115246772766, -1.0491795539855957, -0.4604116380214691, 0.8199282884597778, -0.640548586845398, 1.1088860034942627, -0.6147794127464294, -0.7164620161056519, -0.7849749326705933, 0.426290899515152, -0.25507286190986633, -0.6606030464172363, 0.23180703818798065, -0.29613739252090454, -1.011967658996582, -0.33126208186149597, 0.7218791842460632, -1.0604448318481445, 0.8631645441055298, 1.7940183877944946, 0.4559742212295532, -0.3148374557495117, 1.3111350536346436, 2.4649665355682373, -0.23211044073104858, 1.2346314191818237, -1.2501928806304932, 0.42763498425483704], [-0.8652480840682983, 0.7391132712364197, 1.183719515800476, 0.38189026713371277, 0.09571903944015503, 0.3604371249675751, 1.1595255136489868, -0.03294198587536812, -0.3877592384815216, 0.9675805568695068, 0.7915638089179993, -0.36348479986190796, -0.16318266093730927, -0.202998086810112, -0.6006139516830444, 0.03488294035196304, 0.007625354919582605, 0.22841911017894745, -1.4488224983215332, -0.5419418215751648, -0.3855496942996979, 0.37566912174224854, 0.4732908308506012, -1.3783111572265625, 0.8015064001083374, -1.3715558052062988, 0.13748133182525635, -0.926385760307312, 0.4347471296787262, 0.38476330041885376, -0.18557366728782654, 0.28023818135261536, 0.9166588187217712, -1.3837260007858276, -0.31727951765060425, -0.3279544711112976, 0.4220888614654541, -0.9455610513687134, -1.5078587532043457, -1.1762614250183105, -1.6195893287658691, 1.0164062976837158, 0.6491516828536987, -0.9469427466392517, 0.629877507686615, -0.29523128271102905, -1.9108002185821533, 0.26007628440856934, -0.2723427414894104, -0.6301200985908508, -0.04331783577799797, -2.166689157485962, 0.5363090634346008, 0.45740005373954773, -1.1771594285964966, -1.2196696996688843, 0.7859336137771606, -0.04726126417517662, 0.6940903663635254, -0.9191893339157104, -0.43952879309654236, 2.354057788848877, 1.1478937864303589, 1.3257017135620117], [-1.6804946660995483, -3.152897357940674, -1.1408087015151978, -0.5673004388809204, 1.0996904373168945, -2.1096692085266113, 0.2212561070919037, -1.5229994058609009, 0.6648094654083252, 0.05295591428875923, -1.2074499130249023, 0.015523795038461685, 0.022579889744520187, -0.4945355951786041, 0.35504522919654846, -1.3739937543869019, 0.02607632987201214, 1.5250496864318848, 1.3916646242141724, 0.2135077267885208, -1.4705570936203003, 0.5086724162101746, 1.2746038436889648, 0.10963206738233566, 1.4611537456512451, 0.06470615416765213, -1.7447103261947632, -2.5322086811065674, 0.17886726558208466, -0.10529009997844696, -0.38763612508773804, 0.2991604506969452, 0.11784573644399643, 0.6831347942352295, -0.41845932602882385, 0.3416641056537628, 1.7415738105773926, -0.9439941048622131, -0.20035380125045776, 0.3895794451236725, -0.37944701313972473, -0.4251832067966461, -0.44041866064071655, -1.191183090209961, -0.17614784836769104, -0.2657584547996521, 1.1857200860977173, 1.5062875747680664, 1.7430360317230225, -1.1975338459014893, -0.32158395648002625, 1.3660496473312378, 0.5571195483207703, 0.9557284712791443, -0.27410396933555603, 0.7945915460586548, 1.0644807815551758, -1.6679812669754028, 0.0013689709594473243, 0.7308313846588135, 0.17965896427631378, -1.578641414642334, -3.053954601287842, -0.4818708300590515], [1.682754635810852, 2.212146282196045, -0.30289730429649353, -0.34334370493888855, -0.361602246761322, -1.2018908262252808, -0.17973999679088593, 0.9042803645133972, -1.0996389389038086, 0.02938026748597622, 0.5647199749946594, -0.9544088840484619, 1.1988129615783691, 0.31137725710868835, -0.14144885540008545, 1.09586763381958, 0.42570048570632935, 0.6444944739341736, 0.9913758635520935, 1.1028815507888794, 1.124673843383789, -1.020828366279602, 1.2197270393371582, -0.7230624556541443, -0.017632070928812027, 0.23165263235569, -0.5579929947853088, -0.9744977355003357, -0.27990594506263733, 1.1639961004257202, -0.32428842782974243, 0.5682884454727173, 0.6927895545959473, -1.41562020778656, -2.085487127304077, 0.20767728984355927, 1.984714388847351, -0.3029533326625824, -0.3853628933429718, 0.11797523498535156, 0.11655981093645096, 0.382377028465271, 0.4793355166912079, 0.2767466604709625, -1.6829534769058228, -0.41848909854888916, -0.8632082939147949, 0.3960700035095215, 0.7296227216720581, 0.5604521632194519, 1.4414466619491577, 0.07749596983194351, 0.9429630041122437, -0.4454810321331024, 0.942125141620636, -0.12273348867893219, -0.14751729369163513, 2.070256471633911, 0.23758219182491302, -0.7981696724891663, -0.6109930872917175, 0.15753479301929474, 0.7618745565414429, -1.1548010110855103], [-0.5635707378387451, 1.8702493906021118, -0.7197026610374451, 0.3623889088630676, 0.15542860329151154, -0.4298267960548401, 1.1425691843032837, 1.3189735412597656, -0.2217770218849182, 1.208370566368103, -0.278715580701828, 1.1223211288452148, -0.9086353778839111, -0.4407300651073456, 0.7188849449157715, -1.6387608051300049, 0.9791265726089478, 1.3259425163269043, 0.34901416301727295, 0.2154664397239685, 1.4861886501312256, 2.7695229053497314, -0.23895655572414398, 0.2451733946800232, 1.4920628070831299, 0.8206266760826111, -0.05291979759931564, -1.86098051071167, 0.25537893176078796, 0.5874199867248535, 0.7600371837615967, 1.7433686256408691, 0.42798352241516113, 1.3407706022262573, 0.8220711350440979, 0.6453783512115479, 0.8220216631889343, -0.25028279423713684, -0.3453370928764343, 0.2960900366306305, -2.4438846111297607, -0.3281276524066925, 0.6476980447769165, 0.6169573664665222, 1.2066001892089844, 1.988218903541565, 0.6194916367530823, -0.19864100217819214, -1.993424415588379, 0.25572115182876587, -1.1710572242736816, 0.9139864444732666, 1.3320344686508179, 0.4473285675048828, -1.192151665687561, 1.5013892650604248, 0.028984911739826202, -1.1804991960525513, 0.9711806774139404, -1.4300788640975952, 0.15308016538619995, -0.7910900712013245, -0.9828044772148132, 0.33220040798187256], [0.9019631743431091, 1.311219573020935, -0.7770437598228455, 0.3028338551521301, -0.4867880046367645, 0.005956258159130812, -0.1836155354976654, 0.6827078461647034, 1.021811604499817, -0.2748212516307831, -1.0944316387176514, 1.4744476079940796, -0.9720916152000427, 0.5641038417816162, -1.3624342679977417, 0.3550841510295868, 0.7133359313011169, 1.3722771406173706, 0.000514261715579778, 1.7549400329589844, 0.8012470006942749, -0.029521562159061432, 0.6803059577941895, 0.38884568214416504, -0.258683979511261, -0.5660558938980103, -0.9158076047897339, 2.5844335556030273, -0.3953811824321747, 0.21409067511558533, -1.4232388734817505, 0.19437925517559052, 2.991204261779785, -0.41430556774139404, 0.20765551924705505, -0.1937173455953598, 1.1892997026443481, -0.27733850479125977, 0.354369580745697, -1.1670485734939575, -1.2257003784179688, -0.267631858587265, -1.3452321290969849, -0.5759763121604919, 0.37343254685401917, -0.6251603364944458, 0.4445660710334778, -2.985692262649536, 1.448485255241394, -0.537734866142273, -0.5427529811859131, 1.1446698904037476, -0.21045295894145966, -0.6882543563842773, -0.3533003628253937, 0.7050967216491699, -0.6231066584587097, -0.5095950961112976, -0.9958826303482056, -1.9164459705352783, 0.230333611369133, -0.06436023861169815, 0.39906078577041626, 0.2537032961845398], [0.04387430101633072, 2.2722182273864746, 0.7399759292602539, 0.8906664252281189, 1.139204740524292, -0.20520849525928497, 0.4314993619918823, 0.495265394449234, 0.9367303252220154, 0.6777569651603699, -0.030527370050549507, -1.5790162086486816, -1.8848000764846802, -1.4426380395889282, 0.7786098718643188, -0.2889201045036316, 0.04622514918446541, -1.1360745429992676, 0.2336331158876419, -0.16250315308570862, -2.3063852787017822, 0.7834453582763672, 1.4994714260101318, -0.22416886687278748, 1.7418076992034912, 0.29846498370170593, -1.075990080833435, 2.5423500537872314, -0.3513398766517639, 0.16013318300247192, -0.1546594798564911, 0.600395679473877, -1.3887178897857666, 1.2842305898666382, 0.4120870530605316, -0.3820173740386963, -0.8456928730010986, -0.21133457124233246, 0.0984756276011467, 2.098047971725464, 0.10081622004508972, -0.5623498558998108, 0.41972702741622925, -1.0200361013412476, -2.2067155838012695, 1.2350622415542603, -1.1630775928497314, -0.030453337356448174, 0.6408959031105042, -1.0083047151565552, 1.382176399230957, -1.5373072624206543, 0.24668723344802856, 1.0908902883529663, -1.1434720754623413, 0.7618523836135864, -1.004346251487732, 0.4332701861858368, -0.35097962617874146, 0.08553461730480194, -1.3451943397521973, 1.2098641395568848, 2.050554037094116, -1.092607021331787], [0.3680207133293152, 0.8365403413772583, 0.946262001991272, 0.22726525366306305, 0.0001427125243935734, 1.1325366497039795, -0.439469575881958, -0.2425549179315567, -0.4068797528743744, -1.1572927236557007, -0.19146156311035156, 0.7904029488563538, -0.09611524641513824, -0.47320979833602905, -0.5200279355049133, 0.3352791666984558, -1.3106316328048706, 1.7980740070343018, -1.2985386848449707, 0.7412920594215393, -0.38012629747390747, -0.08757700771093369, 1.6358321905136108, -0.07906035333871841, -0.6988280415534973, -1.2381409406661987, -0.4421134889125824, -1.28870689868927, 0.3055499792098999, 0.07698500156402588, -1.1113899946212769, -0.2655085623264313, -0.010170672088861465, 0.509190559387207, -1.7524104118347168, -1.5627669095993042, -0.3924092948436737, 0.7068293690681458, -0.8880561590194702, 0.695144772529602, 1.2617722749710083, -0.016013823449611664, 0.5545030236244202, 0.7917123436927795, -0.37960442900657654, -0.15530148148536682, 1.4172552824020386, -0.2294941246509552, 0.7769478559494019, -0.5561438798904419, 0.34248945116996765, -1.1418302059173584, 1.2133519649505615, -1.4420431852340698, -1.1256821155548096, -0.025156226009130478, 0.09503275901079178, -0.9495027661323547, -0.8624967336654663, 0.35529592633247375, 0.9472021460533142, -1.3038020133972168, 0.31343477964401245, 0.8323270678520203], [1.2300406694412231, -0.628635585308075, 1.641534686088562, -0.7745152711868286, 1.3599133491516113, 0.25804051756858826, 1.7033885717391968, -1.891822338104248, 0.8646752834320068, 0.5442960262298584, 0.13421256840229034, -0.7122972011566162, -0.4503900706768036, -0.24059219658374786, 0.260744571685791, 0.2799121141433716, -1.4952125549316406, -0.30798760056495667, -0.47235244512557983, 0.08472707867622375, 0.7710580229759216, -0.8516457676887512, -0.7893481850624084, -0.3768438398838043, -1.1890127658843994, -0.5207237005233765, -0.3436317443847656, -0.30728912353515625, -0.0008445254643447697, -0.8609775304794312, 0.22520510852336884, 0.2326272577047348, -1.069674015045166, 0.7921463847160339, 1.1014373302459717, -0.5606333017349243, -0.9852582812309265, 0.5896397829055786, 1.831824779510498, -0.40404388308525085, 2.136394500732422, 1.3755981922149658, -0.33925655484199524, -1.2259163856506348, -0.6952096223831177, -0.21404533088207245, 0.20317450165748596, 0.19501245021820068, -0.832448422908783, 0.5687592029571533, 0.6379779577255249, -2.0942394733428955, 0.32619595527648926, 0.6548250913619995, 2.2173995971679688, 0.8309176564216614, 0.2717140018939972, 0.20195582509040833, 0.9256696105003357, 2.24796724319458, 0.8592053055763245, 0.09511405229568481, -0.5370869040489197, -0.12920792400836945], [-0.5904580354690552, 1.6384531259536743, -0.9097630381584167, 1.435442328453064, 0.5496348738670349, -1.5424928665161133, -0.011767631396651268, 1.6163653135299683, -0.3075610399246216, -1.797006368637085, 0.7439740300178528, -0.48452842235565186, 0.34386852383613586, 0.3633321523666382, 0.5814844369888306, -1.2633883953094482, -0.6666545271873474, -0.24159552156925201, -0.9614680409431458, -0.7396720051765442, 0.48081982135772705, 0.23513847589492798, 1.191497802734375, -0.04701501876115799, -0.6440588235855103, -0.42056238651275635, -1.2751187086105347, 0.12587884068489075, -1.7341023683547974, -0.718644380569458, 1.0408002138137817, -0.09476248919963837, -2.1488800048828125, -0.3814397156238556, -0.9968048334121704, 1.7035936117172241, 1.0869832038879395, 1.895066738128662, 0.5926220417022705, -0.7595646977424622, 0.10263441503047943, 0.08252061158418655, 1.0556460618972778, 1.237369418144226, -0.7100471258163452, -1.107530951499939, -0.4410567283630371, 0.9388287663459778, -0.7120885252952576, 0.14869843423366547, -0.5832706689834595, -1.0723261833190918, -0.28715914487838745, 0.09698992222547531, -1.2109758853912354, 0.07277385890483856, 0.30456700921058655, -0.9529420733451843, 1.0542508363723755, -0.4334697425365448, -0.48227596282958984, -0.7400792837142944, -0.894012987613678, 1.3461459875106812], [0.023653287440538406, 1.7115716934204102, -1.3859341144561768, 0.15334653854370117, -0.9684896469116211, -1.2122541666030884, -0.633916974067688, 0.238794207572937, -0.8619124889373779, 1.6117244958877563, -0.44666802883148193, -0.5360032320022583, 0.24409671127796173, -1.1617919206619263, -1.4036885499954224, -1.2567530870437622, 0.4932006299495697, -1.5439573526382446, 0.00043913291301578283, 0.17223311960697174, 1.3113205432891846, -0.2993796169757843, -0.06532129645347595, -0.2699050307273865, -0.010057883337140083, 0.9629943370819092, -1.3632062673568726, -0.15566125512123108, -0.973849892616272, -2.0072576999664307, -0.042814016342163086, 0.2937766909599304, 0.835234522819519, -0.34260573983192444, 0.02992962673306465, -0.32163262367248535, 1.488176941871643, 0.33301761746406555, 0.43069037795066833, 1.502291202545166, 1.2582497596740723, -1.960568904876709, -1.6294664144515991, 1.2124009132385254, 0.13080282509326935, -0.4061926007270813, -0.6835177540779114, -1.509947419166565, -2.967527389526367, -0.746094822883606, -0.7352805733680725, 0.10573356598615646, -0.785102367401123, -0.8392166495323181, -0.08861050009727478, -1.1124223470687866, -1.3282767534255981, -0.9029719233512878, 0.5574637055397034, 0.6906073689460754, 0.5829927325248718, 0.9878665208816528, -0.4573144018650055, -0.048213813453912735], [-1.5929114818572998, 0.9644786715507507, -0.8230887055397034, 0.8113454580307007, -1.4540696144104004, 0.7324192523956299, 1.3172814846038818, -1.01404869556427, -2.715113401412964, 0.0008077776292338967, -0.2230677604675293, 0.9604515433311462, -0.06591268628835678, 0.39886343479156494, -0.17751429975032806, -0.8608519434928894, 1.1618398427963257, 0.40226948261260986, -1.1343300342559814, -0.5418428182601929, -0.2816428542137146, 0.1649651825428009, 0.5526556372642517, 1.2239149808883667, 0.8136233687400818, -0.36031532287597656, 1.012269377708435, 1.3648754358291626, -1.4275211095809937, -1.1639535427093506, 0.6759263277053833, 0.839415967464447, 1.7975294589996338, -0.12809310853481293, -0.8869360685348511, -0.5782423615455627, -1.5952421426773071, 0.5325091481208801, -1.0595548152923584, 0.5057823061943054, 0.6721192598342896, -0.041717588901519775, -0.19431248307228088, -0.3262045085430145, 0.2258944809436798, -0.7893384695053101, -1.2168809175491333, 0.6857523322105408, 1.857019066810608, -0.38086217641830444, -1.0271072387695312, -0.12548546493053436, -1.5372203588485718, 0.2999403774738312, -2.7052979469299316, 0.1942286193370819, -0.8617305755615234, -0.4967387318611145, -0.07344090193510056, -0.8351190090179443, -1.3709371089935303, 0.2358737289905548, 0.588012158870697, -0.26889896392822266], [-0.44997483491897583, 0.7846388220787048, 0.4788375794887543, -2.212937116622925, 0.20302775502204895, -0.5017099976539612, -0.2940398156642914, -0.15375222265720367, -1.2930757999420166, 0.3700680136680603, -1.0721441507339478, -1.003975510597229, 0.9156702756881714, 1.416980266571045, 0.12337960302829742, -1.2728227376937866, 1.1405659914016724, 1.0243568420410156, 3.731187343597412, -0.6750349402427673, -0.48325785994529724, 1.7729811668395996, 0.9136713147163391, -1.6367250680923462, 0.5678045749664307, -0.7842383980751038, 2.1676740646362305, -0.5825827717781067, -1.1460378170013428, -1.7127779722213745, -0.06672236323356628, 0.5960658192634583, 0.12638737261295319, -1.007271409034729, 0.6604415774345398, 0.26555949449539185, -1.438920021057129, 0.6561828255653381, -0.4220457971096039, 0.5581751465797424, -0.6666528582572937, -0.8821931481361389, -1.225926160812378, -0.04565904289484024, -0.5928547382354736, 0.08242490142583847, -0.2118142992258072, 0.4608093202114105, -0.009528915397822857, 0.8696443438529968, 0.9974980354309082, -0.7382119297981262, 3.0951812267303467, 0.6292104125022888, 0.05571519955992699, 1.198750376701355, 0.05509839579463005, -0.02699020318686962, -0.37195372581481934, 2.011777639389038, 0.5841077566146851, -0.9764494895935059, 0.961679220199585, 0.919133722782135], [0.5938646197319031, -1.6632272005081177, -0.41982004046440125, 0.5969165563583374, -1.4198962450027466, 0.8867263793945312, 0.28694310784339905, 0.16015291213989258, -0.4744868576526642, 0.37357839941978455, -0.6050817966461182, 1.5070321559906006, 0.2819245755672455, -0.8618212342262268, 0.9750549793243408, -0.9420921206474304, 0.58028644323349, -1.9495269060134888, -0.10417267680168152, 1.451128363609314, -1.2110261917114258, 0.6175435781478882, 0.9711543321609497, -2.6725027561187744, -0.25757667422294617, -0.08857713639736176, 0.6852137446403503, 0.9715169668197632, 2.7529876232147217, 0.16253723204135895, -1.685600996017456, -1.0076146125793457, -0.4224085211753845, 0.8872661590576172, 0.2855803966522217, 0.31558287143707275, -0.3076011538505554, -0.3205892741680145, -0.7920780181884766, 0.6146777272224426, -0.4546065926551819, -0.40414831042289734, 1.0388329029083252, 0.16424748301506042, -0.42942342162132263, -0.29882311820983887, -1.5290237665176392, -0.1174682229757309, 0.159819096326828, -0.20243807137012482, 1.4305824041366577, 0.598964512348175, -0.8195500373840332, -2.3312301635742188, 0.31114843487739563, -1.6159067153930664, 0.41762620210647583, -1.9411791563034058, -0.8829862475395203, 0.7551442384719849, -0.9153085947036743, -0.6400018930435181, -0.5041741728782654, -0.8399456143379211], [-1.3878772258758545, 0.8412372469902039, 1.1985158920288086, -0.07997246831655502, 0.879618227481842, 0.19176679849624634, 0.6986461281776428, -0.3899970054626465, -0.08172960579395294, -0.030472006648778915, 0.4554717540740967, -0.4819207191467285, 0.7044816613197327, 0.7560977935791016, 0.5951035618782043, -0.7263365387916565, -0.2985759377479553, -1.6659746170043945, 1.8328790664672852, -0.3981109857559204, 0.3919896185398102, -0.8620503544807434, 0.7504153847694397, -0.29854458570480347, -1.2209651470184326, -0.5248600840568542, 0.4140060245990753, -1.5746179819107056, -0.5238906145095825, -1.7109142541885376, 0.2901577055454254, 0.5048930644989014, 1.7276524305343628, 0.8954474925994873, -0.09845683723688126, -1.4659737348556519, 1.6731480360031128, -0.21157485246658325, 1.199593424797058, 0.9858355522155762, 0.6350880861282349, 0.5159769058227539, -0.05981440469622612, 0.00869232788681984, 0.36950990557670593, 0.3817533850669861, -0.06930272281169891, 1.0291907787322998, 0.7929224371910095, -0.9343039989471436, -0.6422049403190613, 2.8502562046051025, -1.7883925437927246, 0.18577298521995544, -1.045022964477539, 0.821479320526123, -0.03268567472696304, -0.06169578433036804, -0.17415668070316315, -0.38435038924217224, 0.46984097361564636, -0.13348113000392914, 1.4191075563430786, -0.15247692167758942], [0.3146233558654785, 1.0286842584609985, -2.528757333755493, 0.5816653966903687, -0.10484530031681061, -0.5424157977104187, 0.19205139577388763, 0.5252337455749512, 1.6348814964294434, -0.8285307884216309, 1.1479690074920654, 0.8628133535385132, -0.9809234738349915, -0.3471555709838867, -0.450712651014328, 1.0457725524902344, 1.3757694959640503, 1.4906835556030273, 0.3118070363998413, 0.431480348110199, 0.23474213480949402, -0.840777575969696, -0.932991623878479, 1.4570077657699585, -1.5390795469284058, -0.7657201290130615, -1.71885347366333, 0.3248201906681061, 0.06895935535430908, -0.03456214442849159, -0.5856006741523743, 0.7558664083480835, 0.07674351334571838, -1.525529384613037, 0.36867600679397583, -0.7928317785263062, 0.007311693858355284, -1.4909476041793823, -0.206201434135437, 0.4780542850494385, 0.41395026445388794, 0.6769238114356995, 0.6355597972869873, -0.6789997816085815, -0.6431792974472046, 0.7240254282951355, 0.14091181755065918, 1.366926908493042, 1.743807077407837, 1.493119716644287, 0.9268284440040588, 0.5030237436294556, -0.5689817667007446, -0.6220845580101013, 0.2798989415168762, 0.4418746829032898, -0.22022296488285065, -1.0991079807281494, -0.9236041307449341, -0.11423762887716293, -0.5609240531921387, 0.38098037242889404, -1.0486116409301758, 0.9080744385719299], [-0.526958703994751, -0.2791350483894348, 0.6200947165489197, 1.352092981338501, -0.4025489091873169, 0.99675053358078, -1.1915684938430786, -0.3186427652835846, -0.36588776111602783, 0.5042256712913513, 0.22691993415355682, 0.504912257194519, 2.1933467388153076, 1.5291860103607178, -0.8022843599319458, -0.5311782360076904, 1.1368447542190552, 0.16454023122787476, -1.076396107673645, -0.275234192609787, -0.6966677308082581, 0.8983900547027588, 1.3164267539978027, -0.9678886532783508, -0.7457391619682312, 1.284751057624817, -0.9118689298629761, 1.3898261785507202, -2.1531264781951904, -1.9180974960327148, -0.977902889251709, 0.4188380539417267, -0.18362948298454285, 0.47511473298072815, -0.7652811408042908, 1.0930800437927246, 0.2553856074810028, -0.6733309626579285, -0.17546918988227844, 1.9355556964874268, -0.16048692166805267, -0.691948652267456, 1.8467340469360352, 1.4872667789459229, -0.6968844532966614, -0.21550115942955017, -1.0883946418762207, 0.1344982087612152, 1.3549915552139282, -0.07213892787694931, -0.572544276714325, 0.9817273020744324, -1.2946916818618774, 0.5713343024253845, 1.243861198425293, -1.2703516483306885, 0.28765153884887695, 0.298129141330719, 0.05596480518579483, -0.25337931513786316, -2.30940318107605, -1.3335490226745605, -1.4603818655014038, -0.3171210289001465], [3.795989513397217, 0.3374747931957245, 0.28500229120254517, -0.6071994304656982, 0.8701618313789368, -0.5845056176185608, -0.93817538022995, 0.8713531494140625, 0.917456865310669, 0.2948489785194397, 0.2804739773273468, 0.9169226884841919, 0.4742412269115448, -0.11085285991430283, -0.8041155338287354, 0.036662738770246506, -0.2192303091287613, -0.5521686673164368, -0.549758791923523, 0.1867423951625824, -0.7348352670669556, 0.35703879594802856, -0.5983168482780457, -0.6515229344367981, -0.4925652742385864, 0.1842728555202484, -0.4310411214828491, -0.0678027793765068, 0.4379577040672302, -0.7100009918212891, 0.8406877517700195, 0.5153230428695679, 0.09968911856412888, -0.5468512773513794, 2.0384788513183594, -1.3892407417297363, -0.5448344349861145, 1.048225998878479, 0.3565587103366852, -0.05005569010972977, -0.31142646074295044, -1.706071376800537, 0.2865931987762451, -0.8043586611747742, 0.27249130606651306, -0.461172878742218, 0.10856929421424866, -0.736602783203125, -1.0354232788085938, -2.163144826889038, -0.6402466893196106, -0.16611528396606445, 0.38007333874702454, -1.4093108177185059, -0.5344259738922119, -0.8996515274047852, 0.6548526883125305, -0.7572438716888428, -1.937917947769165, -0.5684723854064941, -1.6292076110839844, 0.1129722073674202, -1.1770076751708984, -0.39056944847106934], [0.7032151222229004, -0.4773862957954407, -1.5273690223693848, -0.26470670104026794, -0.08073681592941284, -0.2920398414134979, 0.11354492604732513, 2.5501277446746826, -1.1040332317352295, 0.12411955744028091, -0.32586508989334106, 0.14561448991298676, 1.4022915363311768, -0.0736992284655571, 0.9604646563529968, -0.31982421875, 0.31440284848213196, 1.6273401975631714, -0.30011817812919617, 3.008511543273926, 0.8833035230636597, 0.9359592199325562, 0.0616525374352932, 0.7110973000526428, -0.7600350379943848, 1.0519129037857056, -1.2067041397094727, 1.577962040901184, 0.46977636218070984, 1.6746371984481812, 0.6000553369522095, 0.858092188835144, -0.7590140104293823, -0.6622664928436279, 0.3692781627178192, 1.5982223749160767, -0.5323748588562012, -2.233285665512085, -1.6254169940948486, -0.10918517410755157, -0.43478578329086304, -0.8958848118782043, 1.214349389076233, -1.155716896057129, -0.14281022548675537, 0.0176013745367527, 0.13074904680252075, 1.6610746383666992, -0.596604585647583, -0.7331527471542358, 0.40301886200904846, -0.6700997352600098, 0.6478418707847595, 0.18817833065986633, -1.0634543895721436, 0.8958037495613098, -0.4685859680175781, -0.15313385426998138, 0.8374298214912415, 0.7438604831695557, 1.9297411441802979, 0.9999187588691711, 0.9754278659820557, 1.1980884075164795], [-1.1956859827041626, -1.889487862586975, 2.0109245777130127, 1.4463253021240234, -1.3513989448547363, -2.1039326190948486, -2.760887861251831, 0.07562802731990814, 2.0594282150268555, -1.6291037797927856, 2.649813413619995, 0.5741263628005981, 0.41324296593666077, -1.3426119089126587, -0.056620094925165176, 0.8291376233100891, 1.6372170448303223, -0.43097493052482605, 0.9687392711639404, 0.17632192373275757, 0.5685181021690369, -0.42968225479125977, 0.9776856899261475, -1.2939860820770264, 0.2249305248260498, -1.6317131519317627, 0.49423858523368835, -0.16009029746055603, -0.7783602476119995, 0.019179854542016983, -0.6311310529708862, 0.6845801472663879, -0.34436190128326416, -0.42666879296302795, 0.47520911693573, 0.8708317279815674, 0.6306420564651489, -0.2119135707616806, 0.21162360906600952, -0.7977073192596436, -0.9411895275115967, -0.9840537905693054, -1.393232822418213, 0.5631843209266663, -0.43365997076034546, 1.611624836921692, 0.391433447599411, -0.40207406878471375, -0.4490252435207367, 1.5182223320007324, 0.7040784955024719, 0.9789624214172363, -1.3126448392868042, 1.1526485681533813, -0.9181414246559143, -0.4650937616825104, 1.1515203714370728, -1.0237083435058594, -0.4669938087463379, -0.42167434096336365, -0.8326233625411987, -1.468423843383789, -0.029146980494260788, -0.12085027247667313], [0.5113301277160645, 0.659812867641449, 0.7962725162506104, 1.0926765203475952, 0.26323777437210083, -0.28395456075668335, 0.8484508395195007, -0.35384225845336914, 0.2099159210920334, 1.5771538019180298, 1.6938703060150146, 0.05330260097980499, -0.7043942809104919, 0.18238791823387146, 0.10625401139259338, -1.0647815465927124, -0.2157927304506302, 1.2732142210006714, 0.3843160569667816, 2.0100529193878174, 1.3422212600708008, -0.8021208643913269, 0.41363123059272766, -0.9402076601982117, 1.3260235786437988, -1.070122241973877, -0.07912193983793259, -0.11118140816688538, -0.7300986647605896, 0.9003082513809204, -0.12649047374725342, 0.3964783847332001, -1.5991027355194092, 0.6068856716156006, -0.26927903294563293, 1.08307683467865, 2.1701695919036865, -0.921593427658081, 1.2904725074768066, 0.32077959179878235, -0.49478545784950256, 1.0492784976959229, 0.13118110597133636, -1.7866630554199219, -0.05923394113779068, 0.16665859520435333, 0.4152221977710724, 0.023747997358441353, 1.635185956954956, 0.468485951423645, -0.5886528491973877, 1.0892106294631958, -0.08427438884973526, -0.19722171127796173, -0.5172584056854248, 1.3206560611724854, 1.2726281881332397, -0.7931773662567139, -0.3693488538265228, -1.6210030317306519, 0.4138484001159668, -0.6351271271705627, -1.4503873586654663, -0.591882586479187], [-0.4421830475330353, -2.7459418773651123, -0.0938478335738182, -0.35565686225891113, 2.4880120754241943, 2.8003056049346924, -0.6142589449882507, -0.4293896555900574, 0.3017050623893738, 0.16703902184963226, 0.4239986538887024, -0.27171099185943604, -2.4258949756622314, -0.32885226607322693, 0.5238766074180603, -0.7857568860054016, 1.3137133121490479, -0.2933691442012787, 0.6204188466072083, -0.22496527433395386, 0.6933613419532776, 0.0908823236823082, -1.9206833839416504, -0.2112160474061966, -0.19501005113124847, -0.10220442712306976, 0.4289180636405945, 0.14308348298072815, -0.09718116372823715, -1.3982818126678467, 0.3689238131046295, 0.660573422908783, -0.7358236312866211, -0.30119413137435913, 0.3710879683494568, -1.3881618976593018, 0.9452463984489441, 0.027840645983815193, -0.6163240671157837, -0.6841270327568054, -0.4328608512878418, 0.6006935238838196, -0.6174466013908386, 1.0135977268218994, 0.1022469773888588, -0.6090586185455322, -0.5156654715538025, 1.0696258544921875, -0.5045390725135803, -0.18410548567771912, -0.7649127840995789, 0.34170812368392944, 0.326770156621933, 1.4800047874450684, 0.4554483890533447, 1.0065561532974243, 0.3099420666694641, -0.727066159248352, 0.6325918436050415, 0.2546868324279785, 1.2506486177444458, 0.12715382874011993, -0.7164180278778076, -0.45959725975990295], [1.2291008234024048, -0.6568736433982849, -0.6036596894264221, 1.2259193658828735, 1.4478390216827393, -0.04405617341399193, 0.6543222665786743, 1.4749196767807007, -0.8061808347702026, -0.18749816715717316, 0.3122102916240692, -1.5527863502502441, -0.20974917709827423, 0.4536876976490021, 0.08535882830619812, -3.1771020889282227, 0.6061956882476807, 0.39596793055534363, 0.7278221249580383, -1.1260732412338257, 0.38157087564468384, 0.24566945433616638, -1.8294259309768677, 2.5041563510894775, 1.8817845582962036, 1.2786660194396973, 2.1312310695648193, 0.1365835815668106, -0.4244745075702667, 1.3274915218353271, -1.601344347000122, 0.24974197149276733, -0.7081591486930847, -0.7053951025009155, 0.725879967212677, 0.5000538229942322, -0.7434505820274353, 1.3035227060317993, -0.6280537247657776, 0.6428121328353882, -0.9595373272895813, 0.8566159009933472, -0.48949363827705383, -1.570050835609436, -1.0436763763427734, 1.378466010093689, -0.5336247682571411, 0.9359540343284607, 0.00759715074673295, -0.9110943675041199, -0.606774091720581, 0.8777021765708923, -0.2640770375728607, -0.9895147085189819, 2.649282693862915, 0.12419027835130692, -0.505782961845398, -1.2187172174453735, -0.16831615567207336, 3.044919013977051, 1.1213892698287964, 0.2998981773853302, 0.3522050380706787, -1.5387130975723267], [-1.4191988706588745, 0.5888254642486572, 0.4484327733516693, -1.4148550033569336, 2.0024869441986084, -1.7393908500671387, 0.7176840305328369, -0.5802128314971924, -1.4560964107513428, -0.892787516117096, -1.6054937839508057, -1.6834584474563599, 0.20440037548542023, 1.0248581171035767, -1.378680944442749, -0.16340629756450653, 0.022934485226869583, 1.5090256929397583, 1.0777602195739746, 0.25204765796661377, 0.447549045085907, -1.3800060749053955, 1.3073912858963013, 0.6095348596572876, 0.8940474987030029, 0.0855160653591156, -1.137536883354187, -0.7389626502990723, 0.320665180683136, 0.41431066393852234, 0.9927083849906921, 0.08436177670955658, -0.31467464566230774, 0.48202937841415405, -1.158474087715149, -0.4938325881958008, 1.2136099338531494, 1.2119625806808472, -0.6013422608375549, -0.24654774367809296, 1.3999680280685425, 0.8330060243606567, 0.9035482406616211, 0.01825680211186409, 0.3126320242881775, -0.6638379693031311, -1.1701157093048096, 1.3957048654556274, 0.6704370379447937, -0.9265533685684204, -0.4101523756980896, 1.0117675065994263, -0.8321603536605835, 2.3632938861846924, -0.6491805911064148, 1.6484445333480835, -1.101061463356018, -0.04029775410890579, -0.042174264788627625, 0.4505101442337036, -0.23694466054439545, -0.6326223611831665, 0.625972330570221, -0.8054684996604919], [-0.7038221955299377, -1.9018787145614624, 1.4791924953460693, 0.4995848834514618, -0.04411109909415245, -0.08318427950143814, 0.9134040474891663, 1.052537202835083, -0.7178254127502441, 1.4949607849121094, -0.002940910402685404, -0.34691768884658813, -1.5793052911758423, -1.8671780824661255, -0.6344183683395386, 1.1630085706710815, 0.10775814950466156, 0.12005062401294708, -0.6773648858070374, -0.8569055199623108, -0.9545539617538452, -0.5576344132423401, 0.5583128333091736, -0.0013822219334542751, -0.12965618073940277, -0.16527201235294342, 0.41299429535865784, 0.61548912525177, -0.5026164054870605, 1.0739588737487793, 0.11842688918113708, -0.25295019149780273, -0.21700944006443024, -0.5052104592323303, 2.5704987049102783, 0.27857422828674316, 1.9390366077423096, 0.049429185688495636, 0.3795482814311981, -1.1017451286315918, -0.5856372714042664, 1.7236623764038086, 0.6767197847366333, -1.767993450164795, 0.7119781970977783, 1.0732589960098267, 0.2691561281681061, 0.978125810623169, 1.1631263494491577, -0.32215479016304016, 1.1625535488128662, -0.8418889045715332, 0.485233873128891, -1.4360631704330444, -0.46908631920814514, 0.5844370126724243, -1.0191824436187744, -1.8977982997894287, -1.1635233163833618, -1.1814155578613281, -1.0147422552108765, 1.8574082851409912, 1.0898072719573975, -0.6226929426193237], [0.36637935042381287, 0.17929419875144958, 0.5169008374214172, 0.5063714385032654, 0.8934516906738281, -0.6450340747833252, -0.22339394688606262, 0.03750200569629669, -0.4191508889198303, 0.07880846410989761, 1.2602084875106812, -0.6250848174095154, 0.18645183742046356, -0.02491477131843567, -1.2296558618545532, -0.06398593634366989, 0.33189019560813904, 0.7065140008926392, 1.4620592594146729, -0.15159596502780914, 1.074167251586914, -0.5907477140426636, 2.4435369968414307, 0.2721218168735504, 0.014002147130668163, -1.1817936897277832, 0.6463683843612671, 0.6082417964935303, -0.005936221685260534, -0.6257556676864624, 0.037295032292604446, 0.5900837779045105, -1.5351883172988892, 0.3072430491447449, -0.09606119245290756, -1.844892978668213, -1.7113146781921387, -0.20472894608974457, 0.6140897274017334, -0.6571623682975769, -0.5356003046035767, -0.33561933040618896, -0.2221912145614624, 0.11024373769760132, -0.5737621784210205, 1.3163928985595703, -1.1012351512908936, 0.6757991313934326, -1.5802170038223267, -0.5383710265159607, -0.11024235188961029, -0.6386392116546631, -1.937361717224121, 0.3032930791378021, 0.05912162736058235, 0.6111261248588562, -1.0348855257034302, 1.741517186164856, 0.47824978828430176, -0.03764737769961357, 0.016517870128154755, 0.9716033339500427, 0.05566268786787987, 1.1248321533203125], [0.15020941197872162, -0.7378228902816772, -0.39015480875968933, 1.0089592933654785, -0.6352500319480896, 0.9213342070579529, 1.0005979537963867, -0.1271141618490219, -0.6998913288116455, 0.17139160633087158, -1.0083001852035522, -0.6716009974479675, -0.24933172762393951, 0.4507095217704773, -0.7478213906288147, -0.6746188998222351, 1.8304311037063599, 1.313308835029602, -0.02157001569867134, -0.013902237638831139, 1.0212582349777222, 0.0802488699555397, 0.07837697118520737, 0.14255885779857635, 1.8822588920593262, 1.1592072248458862, -0.13759467005729675, 0.18266992270946503, -0.3457617163658142, -1.3848897218704224, 0.8890224099159241, 0.63127201795578, 0.714676022529602, -1.2459535598754883, 2.0929489135742188, 0.8158999085426331, -0.39432963728904724, 0.4536551535129547, -1.6533066034317017, 0.09890265017747879, 1.0178903341293335, -0.29738637804985046, 0.022163379937410355, -0.8367326855659485, -1.2919596433639526, -1.6542848348617554, 1.3342742919921875, -1.3258894681930542, -0.2186887115240097, -0.22636744379997253, -0.025737980380654335, -1.0298696756362915, -1.086045265197754, 0.7383570671081543, 0.5993362069129944, -0.04431862384080887, -0.9132372140884399, -0.3999338448047638, -0.3463800847530365, 0.35828644037246704, -0.5350553393363953, 0.8333895206451416, 1.431027889251709, 0.9872922301292419], [-0.19370931386947632, -0.5034657120704651, 0.22893445193767548, -0.8357468843460083, 0.028934869915246964, 0.7852180004119873, 0.46885791420936584, 0.24038928747177124, 0.5532175302505493, 0.6291982531547546, -0.8581041693687439, 0.8722197413444519, 0.38055869936943054, 1.536730170249939, -1.4837770462036133, -0.5323424339294434, 2.0319528579711914, 0.3913463056087494, 0.6414833664894104, -1.1757521629333496, -0.43831685185432434, -0.5172014832496643, 1.274561882019043, 0.384078711271286, -1.2713677883148193, -0.5439749956130981, 0.7644760012626648, -0.5564209222793579, -0.046483609825372696, 0.7733196020126343, 1.2144056558609009, -0.3219873607158661, -0.5271133780479431, -0.1366523951292038, 0.8338894248008728, -0.8074827194213867, 1.461897850036621, 1.0234441757202148, -0.5003772377967834, 0.6196072697639465, -0.2808622121810913, 1.845229148864746, -0.3237762749195099, -0.670796275138855, 0.4374741017818451, 0.049918513745069504, 0.39199715852737427, -1.048693299293518, 0.990511953830719, -0.27561047673225403, 1.2197723388671875, 0.6775432825088501, 0.5533846020698547, -0.1934417188167572, -0.05650484561920166, 0.08356646448373795, -0.47275376319885254, 0.5441834926605225, 2.1850926876068115, -0.06229768693447113, -2.0589756965637207, 1.9235870838165283, -0.938082218170166, -2.2283554077148438], [-0.21275514364242554, 0.08401690423488617, 0.08671624213457108, -0.4882107675075531, 0.2989985942840576, -0.37372148036956787, 1.4536916017532349, -0.03993125632405281, -0.3078581690788269, -1.0642709732055664, -1.0450892448425293, -0.6740618348121643, 0.8613818287849426, -0.6856629252433777, 0.943801999092102, 1.6254149675369263, -0.47206056118011475, -0.37235963344573975, 0.6504835486412048, 1.6037718057632446, -0.76653653383255, -0.17657184600830078, 0.9801360368728638, -0.08401022106409073, 0.813115119934082, -0.054516054689884186, -0.005965528078377247, -1.2242493629455566, -0.8558030724525452, 0.40532588958740234, 2.6204631328582764, 0.3394869863986969, -0.6562098860740662, 0.5525513291358948, -1.2598609924316406, 0.669340193271637, -0.09833241254091263, -1.104490876197815, 0.05463798716664314, 0.7851778864860535, -0.13252393901348114, -1.5795115232467651, 0.9857056140899658, -0.7776681184768677, 0.2135239690542221, -1.0463858842849731, -0.5117974877357483, 1.2162765264511108, -0.3001329302787781, 0.1606445014476776, 0.5637761354446411, -1.3637477159500122, -0.12461258471012115, -1.854402780532837, 0.2778104841709137, 0.1665836125612259, -0.3813026547431946, 0.5275181531906128, 0.4437238872051239, -0.5735381245613098, -0.48141786456108093, -0.39545539021492004, -0.4793255031108856, -1.8077377080917358], [-1.162888526916504, 2.1727371215820312, 0.6562421321868896, -0.7268426418304443, 0.3153490722179413, -1.7180544137954712, -1.0744740962982178, -0.9900678992271423, -0.14221718907356262, 0.49489298462867737, -0.14583472907543182, 0.36661964654922485, -0.21687017381191254, 0.23871368169784546, 0.5498981475830078, 0.514154851436615, 1.6919176578521729, 0.3469506800174713, -2.106092929840088, 2.079617500305176, 0.8342159390449524, 0.36189979314804077, 0.13735944032669067, 0.9598891139030457, -0.6503961086273193, 0.4716401696205139, 1.7104238271713257, -0.10179470479488373, 0.2959549129009247, -1.2656471729278564, 0.11625678092241287, -0.6746400594711304, -0.6467960476875305, -0.39062798023223877, -0.555176317691803, 0.29592031240463257, -1.2766399383544922, -0.907555103302002, -0.2800309360027313, 0.5804961919784546, 0.8775357604026794, -0.17957617342472076, -1.062562108039856, 0.503244161605835, 1.1226407289505005, -1.230879545211792, 2.4828362464904785, 1.3130682706832886, -0.29933372139930725, 0.2134850174188614, -0.01327363308519125, 0.8805686235427856, 1.4956456422805786, 0.09654268622398376, 0.17764276266098022, -0.33232352137565613, 0.06159855052828789, -0.17222386598587036, 1.307511568069458, 0.13418696820735931, -1.4200109243392944, -1.7113929986953735, -0.9891586303710938, -0.6530721783638], [0.9890460968017578, -0.2675827741622925, -0.1693732738494873, -1.2173025608062744, 0.4089762568473816, 1.15268075466156, 0.2789601981639862, -0.07008658349514008, -0.3847573697566986, -0.7552400231361389, -1.5151981115341187, -0.3633391857147217, -0.3256356716156006, -1.9337438344955444, -0.2941346764564514, 0.32838767766952515, -0.4240656793117523, 1.5152491331100464, -1.3970633745193481, -0.8394591808319092, -1.4229061603546143, -0.5279004573822021, -0.9628017544746399, -0.8302609324455261, 0.22747229039669037, -1.6241470575332642, 0.70372474193573, -0.14043353497982025, 0.5897637009620667, 0.31172484159469604, -0.2325444370508194, -0.9359950423240662, 1.3314369916915894, 0.9620229601860046, -0.26662397384643555, -0.4841664135456085, -1.165989637374878, -0.7900751829147339, 0.18315953016281128, -0.3734363913536072, 0.10532207787036896, 0.4401567280292511, 1.2539883852005005, -0.0032217155676335096, 3.0856618881225586, 0.8475234508514404, -0.06392087042331696, -1.4598358869552612, -0.9245094656944275, 0.7680934071540833, 0.6619753837585449, -1.3589009046554565, 0.5276776552200317, -1.0657552480697632, 0.009388580918312073, -0.04786837846040726, 0.7850050926208496, -1.3643131256103516, -0.16173607110977173, 0.5344944596290588, -0.12160313874483109, -0.17099520564079285, 0.9085221886634827, -0.5556214451789856], [-1.5913926362991333, -0.0007355137495324016, -1.6715279817581177, -0.19391074776649475, -0.8154336810112, -0.26957544684410095, -1.0240153074264526, 1.207493782043457, 1.1098417043685913, -1.792212963104248, -1.5567948818206787, -0.6448900103569031, -1.3551230430603027, -0.021553169935941696, 0.5379661321640015, -1.0761237144470215, 0.36793240904808044, 0.03566344827413559, 0.24787284433841705, 1.6008137464523315, 0.5328429937362671, 0.08504590392112732, 1.0906524658203125, -0.05934982746839523, -0.22342398762702942, 2.2094027996063232, -0.2040393054485321, -0.9433231949806213, -1.2291215658187866, -1.1757701635360718, 1.3715537786483765, 1.8733234405517578, -1.209189772605896, -0.4821211099624634, -0.7313535809516907, 0.7906298637390137, -0.5623791217803955, -2.9240918159484863, -0.28418081998825073, -0.8270851969718933, -0.3840527832508087, -1.0795502662658691, 0.8916217088699341, -1.7536464929580688, 1.1231001615524292, 0.839285671710968, -0.552250325679779, 0.4643312096595764, -0.6914344429969788, -0.15507549047470093, 0.39090418815612793, -0.6067901849746704, 0.30320626497268677, -0.32997339963912964, 0.06228140741586685, 0.3244379758834839, 0.22369453310966492, -0.048323165625333786, -0.9394922256469727, 0.9734811782836914, -1.1553399562835693, -0.022805744782090187, 1.3109400272369385, -0.26995155215263367], [0.2936577796936035, -1.5241920948028564, 0.03060716949403286, -0.4598044455051422, -0.7603870630264282, 0.7780812978744507, 0.5110015869140625, 1.059711217880249, -0.12692521512508392, 0.28301867842674255, -0.284390926361084, -0.25839555263519287, -0.12317141890525818, -0.6078141331672668, -0.5680011510848999, 0.2730611562728882, -0.3878779113292694, 0.34721043705940247, -0.6343043446540833, -0.2025274783372879, -0.3018099367618561, -0.30876901745796204, 1.0985918045043945, -1.6991779804229736, 2.3644096851348877, -0.14781101047992706, -0.7273560166358948, 0.8367024064064026, -0.47003042697906494, -0.2269640415906906, -0.9052888751029968, -0.9957674741744995, -1.2187769412994385, -0.4715743362903595, 0.9690539240837097, 0.9648880958557129, 0.16529807448387146, 1.267579197883606, 0.5339046716690063, 1.578741431236267, 0.03331556171178818, 7.698722765780985e-05, -0.8755748867988586, 2.217891216278076, 0.16674932837486267, 1.451823115348816, -0.47271162271499634, -0.5401795506477356, 0.891162633895874, 1.346381425857544, -2.3038337230682373, -2.1298437118530273, 0.2914270758628845, 0.5034385919570923, -0.024414431303739548, -0.7864148616790771, 0.5198742747306824, -0.5193665623664856, 0.2165733426809311, 1.734651803970337, -0.9611229300498962, -2.0613744258880615, -0.9807608127593994, 1.8154855966567993], [0.47979551553726196, 0.38789746165275574, 0.4028160274028778, 0.39155957102775574, -0.4613824486732483, -2.577606439590454, -2.1985857486724854, -0.3634885549545288, 1.1418702602386475, 1.5213011503219604, -1.037731409072876, 1.0398503541946411, 1.3320841789245605, -0.03180130943655968, 0.6384535431861877, 1.4128644466400146, 1.3619436025619507, -1.2574867010116577, -0.06061156466603279, 0.4488718509674072, -2.0590906143188477, 0.21758519113063812, 0.08494801819324493, -1.5758826732635498, -1.2995413541793823, -0.7752025127410889, -1.6079076528549194, 0.41789671778678894, -0.6814242601394653, 1.107683539390564, -0.0807209312915802, 1.1167842149734497, 0.5806530117988586, -0.2007220983505249, 0.3743441104888916, 1.4860607385635376, -0.35307541489601135, -0.1363188922405243, 0.25597572326660156, -0.41570737957954407, -0.49058467149734497, -0.8914459347724915, 1.1758277416229248, -2.1442885398864746, 0.5553099513053894, 1.6796849966049194, 0.6178470253944397, -1.351172685623169, 0.8470999598503113, -1.1050997972488403, -0.002128611085936427, 0.7941069602966309, 1.217818021774292, -0.13636192679405212, 0.6447842717170715, 0.20761042833328247, 1.2645004987716675, -0.32143378257751465, -0.5623594522476196, -0.3373442590236664, -1.5756785869598389, -0.02134479023516178, -0.7803744673728943, 0.6861631870269775], [-0.7349327206611633, 1.5440505743026733, -0.35899338126182556, -0.13772746920585632, 0.3863351345062256, 0.13478629291057587, -0.14208917319774628, 0.15199755132198334, -0.16286754608154297, 0.49197548627853394, -0.7953683733940125, -1.5706177949905396, 1.0606462955474854, -0.6165956854820251, -1.3047454357147217, -2.132215738296509, 0.7599660754203796, -1.2599453926086426, -1.9189521074295044, -1.3266363143920898, 0.2730415463447571, 0.21133780479431152, 0.02290908247232437, -1.2891935110092163, -0.7398437857627869, 0.7437253594398499, -0.739494264125824, -0.5985066294670105, 0.7140648365020752, 0.28809815645217896, -2.0475974082946777, -0.7140935063362122, -1.096693515777588, 1.4146265983581543, 0.4000822901725769, -0.11137834191322327, -1.0361686944961548, 1.2597652673721313, 1.7323838472366333, -0.5666401386260986, 0.17203353345394135, 0.29087382555007935, -1.7170982360839844, -2.4563839435577393, -1.7184613943099976, -0.4076891243457794, -0.3171559274196625, 0.5328047275543213, 0.1135466918349266, -2.0171573162078857, 0.5916192531585693, -0.6170826554298401, 0.004467188846319914, -1.4310592412948608, 0.47713884711265564, -0.9572092890739441, -0.2588641345500946, 0.8789864182472229, -0.4710668921470642, 0.2188461273908615, 1.6438803672790527, 0.8192048072814941, 0.4251830279827118, 0.9620477557182312], [-0.2848978340625763, 0.3256520628929138, -0.12722061574459076, 0.9628520607948303, 1.2322825193405151, -1.0329880714416504, 0.19308686256408691, -0.23660238087177277, 0.9839193820953369, -0.293602854013443, 0.38356491923332214, 1.003113031387329, -0.6086695790290833, -1.3867237567901611, -1.0086959600448608, 1.3720523118972778, 0.32108762860298157, 1.7610756158828735, 1.3708070516586304, -1.3140133619308472, -0.7982063889503479, -2.1780807971954346, 0.9696319103240967, -0.01349768415093422, -0.6460121870040894, 0.5399268865585327, 1.4650837182998657, -1.8949981927871704, 1.6426788568496704, -0.19156615436077118, 0.8709704875946045, 0.6716201305389404, -0.26556965708732605, 1.4979338645935059, -0.48179638385772705, 0.7306756377220154, 1.8388422727584839, -0.08259058743715286, 0.8621571660041809, 0.2760274112224579, 1.0043864250183105, -1.398460865020752, 0.2701869606971741, 1.8508927822113037, 1.0751352310180664, -0.3287789225578308, 0.43323028087615967, -0.20910634100437164, -0.8049399256706238, -0.3487582206726074, 0.9938384294509888, -0.08871079236268997, -0.07264377176761627, 1.7854430675506592, 1.0329164266586304, -0.4038914740085602, -0.5111492276191711, 0.7427955865859985, 0.015910418704152107, 0.10066249221563339, -1.661376953125, 1.119226336479187, -0.7652211785316467, -0.989851713180542], [-0.566465437412262, 0.3978101909160614, -0.28751590847969055, 0.5718271732330322, -0.5299780368804932, -0.06522385030984879, 0.19984212517738342, 0.5968297123908997, -0.9698694944381714, -0.20843449234962463, -0.35900673270225525, -0.6094078421592712, -1.1355417966842651, 1.3903329372406006, -2.584057092666626, -0.6543049216270447, 1.5732501745224, 1.8753859996795654, -0.9590527415275574, 0.313923180103302, -0.07053239643573761, -0.6081167459487915, -1.3489898443222046, -0.15287218987941742, -0.5379805564880371, 1.6268295049667358, 1.2455129623413086, -0.4120134115219116, -1.0751490592956543, 0.14025571942329407, -0.7486759424209595, 0.17663182318210602, 1.335585117340088, 2.734130620956421, 0.5611878633499146, -1.7671840190887451, 0.15027713775634766, -0.018132822588086128, 2.00956392288208, -0.3412330746650696, -1.3002480268478394, 1.0533654689788818, 1.6068179607391357, -0.6114599108695984, 0.6693230867385864, -0.30439379811286926, -0.5942936539649963, -0.04473796859383583, 0.23680847883224487, 1.2249385118484497, 0.40121155977249146, 0.27926090359687805, -1.3494129180908203, -0.016567902639508247, -0.33296409249305725, -1.1171802282333374, -0.16060912609100342, -1.5898996591567993, 1.3164939880371094, -0.5942843556404114, 0.5827667117118835, 0.7137641310691833, 0.47960272431373596, 0.12886033952236176], [0.39608630537986755, 0.6139923334121704, 0.4483327567577362, -0.394158273935318, -2.1102590560913086, 0.7759109735488892, -1.563632607460022, -0.4248543679714203, 1.1509073972702026, -1.212783694267273, -1.6309465169906616, -0.293072909116745, 1.2796976566314697, 0.13236157596111298, 0.645449161529541, -0.32552772760391235, 1.6361498832702637, 0.024194978177547455, 1.207999587059021, 0.39414793252944946, 1.211158037185669, 0.21006549894809723, 1.5872066020965576, 0.779051661491394, 1.4876682758331299, 0.12990149855613708, -0.7532792091369629, -0.22796443104743958, 2.11696195602417, -0.5643284916877747, -0.9942877888679504, 0.6151496171951294, -0.5951759219169617, 0.983433187007904, -0.541569709777832, -0.3800415098667145, 0.30074238777160645, 1.2094571590423584, 0.38160157203674316, 0.7942966222763062, 0.8278285264968872, 0.05263719707727432, -0.8763250708580017, -0.9115209579467773, -1.2709145545959473, 0.3657035231590271, 0.041103605180978775, 0.04599011689424515, 0.25175726413726807, 0.6287195682525635, 0.040916528552770615, 0.29185566306114197, 0.3859443664550781, 0.15653426945209503, 0.557694673538208, 1.0783147811889648, 0.6458056569099426, 1.0679799318313599, 0.1256478875875473, 0.6899353265762329, -0.5037151575088501, 0.3059193193912506, 1.2608988285064697, 0.10590330511331558], [-0.1882302463054657, -2.0337424278259277, -0.5042489171028137, 0.9943807721138, 1.5878859758377075, 0.317050039768219, 2.620894432067871, 0.6485146880149841, 0.4650907814502716, -2.2066965103149414, -0.008559362962841988, -0.48902082443237305, 0.4471926987171173, 1.6887383460998535, 1.3869082927703857, 1.3534493446350098, -1.4897541999816895, -0.28288722038269043, 0.07015466690063477, 1.5366005897521973, 1.2697961330413818, 1.334279179573059, -0.02657199464738369, 0.5341615080833435, 0.3199079930782318, -0.3100723922252655, 0.13782387971878052, 1.1402666568756104, 0.1577090322971344, -1.463474154472351, -0.5250275731086731, -0.6207302808761597, -1.046018123626709, 0.26773741841316223, 0.15834137797355652, -1.6232010126113892, 1.08998441696167, 1.1885384321212769, -1.6680673360824585, -1.2788223028182983, 1.3264816999435425, 0.36432531476020813, -0.9234185218811035, -0.5006198883056641, 1.0571612119674683, 1.3133313655853271, 0.2656209468841553, 0.5530254244804382, -1.4946887493133545, -0.6617391705513, -1.741277813911438, 1.1031508445739746, -0.9674757122993469, 1.8307327032089233, -0.4700394570827484, -1.9009149074554443, 0.19325664639472961, 1.620963215827942, -1.064634084701538, 0.4412807524204254, 0.3303924798965454, 0.42088258266448975, -0.7134612798690796, 1.8389638662338257], [1.0087660551071167, -1.3562992811203003, -1.0961799621582031, 0.7123124599456787, -0.14339400827884674, -0.8255075812339783, -0.2094554305076599, -0.32509785890579224, 0.5100933313369751, 0.7833315134048462, 1.0175187587738037, -0.022342657670378685, -0.14333412051200867, 1.7854173183441162, -1.535312294960022, -0.7419497966766357, -0.8721065521240234, 2.083266258239746, 0.45608121156692505, 0.10052517056465149, -1.4528907537460327, 1.8588144779205322, 0.32380056381225586, -0.23035494983196259, 0.6977973580360413, 0.7348509430885315, 0.7215971350669861, 0.8751763701438904, 0.3843493163585663, 0.2481229454278946, 0.6628992557525635, -0.054268740117549896, 0.10359298437833786, -1.0433119535446167, -0.611392080783844, 0.020937640219926834, -1.5835444927215576, 1.291545033454895, -0.08946865051984787, -0.9932661056518555, -1.0595368146896362, 1.1959487199783325, -1.023355484008789, -0.6052089333534241, -0.6324271559715271, 0.3177129030227661, 0.2901679277420044, 0.32160231471061707, -0.3113834261894226, 0.9911991357803345, 1.2546186447143555, -0.9688753485679626, 0.39258769154548645, -0.3744060695171356, 0.04184902086853981, 1.8353644609451294, 0.20564904808998108, 0.28248074650764465, -0.09745755791664124, -0.8802156448364258, -1.1732025146484375, -1.0825453996658325, -1.0126497745513916, -0.4590969383716583], [0.5187665820121765, -0.08996443450450897, -0.6113583445549011, 0.42432889342308044, -0.6729687452316284, 0.025060521438717842, -0.14144586026668549, -0.7345058917999268, -0.10997205227613449, -0.021335287019610405, 0.1675442010164261, -1.8128647804260254, 0.1727212518453598, 1.0670262575149536, 1.2362127304077148, 0.18485841155052185, 0.3330397605895996, -1.1096954345703125, -3.1581766605377197, 0.533645510673523, -0.6534920930862427, 1.2152783870697021, 0.07160536199808121, -0.6026831269264221, 0.9135048985481262, 0.6493158340454102, -0.9076331257820129, 0.9163923263549805, 0.23896801471710205, 0.005151144228875637, 1.173501968383789, -0.06524596363306046, 0.016771726310253143, -0.6098617315292358, -1.3857308626174927, -1.445611834526062, 0.7794597744941711, -1.2561770677566528, 0.8641975522041321, 0.7432569265365601, 1.2418502569198608, 0.19742852449417114, 0.002825898816809058, 0.7905724048614502, -0.30212733149528503, 1.0376471281051636, -0.5328415036201477, 0.8943901658058167, 1.5255322456359863, 0.12235531210899353, 0.8264129757881165, 0.6606956124305725, -0.9576272964477539, 0.2472626119852066, 0.5951366424560547, 1.3208081722259521, -0.7586955428123474, -0.6557093262672424, 2.1401116847991943, -0.7266457080841064, 0.26335445046424866, -2.05515456199646, 0.9635321497917175, -1.08823561668396], [1.2529772520065308, 0.0409090481698513, -1.484276294708252, -0.45596638321876526, -0.2823810875415802, -0.6557227969169617, 1.6935032606124878, 0.06669794023036957, -2.1033647060394287, 1.8477199077606201, -0.07891370356082916, 0.41085970401763916, -3.039759397506714, 0.6708216071128845, -0.8504585027694702, 1.570047378540039, -0.6021199822425842, 2.189711809158325, -0.3648832142353058, -0.18348823487758636, -0.3012528419494629, -1.4661282300949097, 1.1033744812011719, 0.2435608208179474, -0.28996941447257996, -1.3342633247375488, 0.35553646087646484, -0.4663477838039398, 0.1326475739479065, 0.4927526116371155, -1.0564430952072144, -0.43299898505210876, -2.1359527111053467, -0.4177159070968628, -0.17129576206207275, 1.8475359678268433, -1.5252156257629395, -0.44442638754844666, -0.7109712958335876, -1.169987678527832, 1.3099169731140137, -0.14141419529914856, 1.348138689994812, 0.9372423887252808, -0.8438968062400818, -0.742734432220459, 0.8771201372146606, 0.15675251185894012, 0.7312086224555969, -0.3447719216346741, 0.14962081611156464, 0.502078115940094, 1.643826961517334, 0.5039013028144836, -1.910667896270752, -0.15984651446342468, -1.2634403705596924, 2.175384759902954, -1.783084511756897, -0.1009763553738594, -0.2957935929298401, 1.136508822441101, -0.31378230452537537, -1.2619153261184692], [-0.5377653241157532, -0.5640715956687927, -1.5738507509231567, -0.151208758354187, 1.0530446767807007, -0.10079099982976913, -0.9069536924362183, 0.38854092359542847, 2.003621816635132, 0.03799951449036598, 1.016561508178711, -0.6867338418960571, -0.7097092270851135, 0.6770539283752441, 0.657374918460846, 1.5371906757354736, -1.6367828845977783, -1.2070595026016235, 0.8804612755775452, -0.3119102120399475, 1.4661037921905518, -0.29671335220336914, -0.06512816995382309, -0.1889672726392746, -3.110400915145874, 0.9361393451690674, 0.19123198091983795, 2.1900153160095215, -0.7517953515052795, -0.3499104678630829, 0.0476357527077198, -1.0190266370773315, 1.0764915943145752, -1.625127911567688, 0.6366382241249084, 1.7942957878112793, 0.5796172618865967, -0.37385308742523193, 2.2782480716705322, -0.925372302532196, -1.214809536933899, 0.8857806921005249, 0.8935897946357727, -0.6860091686248779, 0.7327218055725098, 1.1136689186096191, -0.6403645277023315, -1.4343068599700928, -1.1667790412902832, -0.5337350368499756, 0.20285563170909882, -1.080925703048706, -1.0855813026428223, -0.30988335609436035, 0.8202682733535767, 1.3978610038757324, 2.266371965408325, 0.06457359343767166, 0.7334016561508179, -1.0441802740097046, 0.027042008936405182, 0.8242174386978149, 1.2032949924468994, -2.015411138534546], [0.5145882368087769, -0.3686390817165375, 0.576332151889801, -0.6955506205558777, -2.554560899734497, 1.4793630838394165, 0.7222597599029541, 0.9056951403617859, 1.4809231758117676, -0.002752351574599743, 0.5732749104499817, 1.7088186740875244, -0.7671082615852356, -0.0859910100698471, -0.03135940060019493, -0.8220503926277161, 1.0231783390045166, -0.9560298919677734, 1.927992582321167, -0.72211754322052, 1.2493690252304077, 0.5918041467666626, 0.7861831784248352, 1.465177059173584, 0.815716564655304, 0.8809346556663513, 0.035393644124269485, 0.5590609908103943, 0.013889068737626076, 1.3988678455352783, -0.5089277029037476, 0.14775310456752777, 1.6814268827438354, 0.30083605647087097, -1.5745490789413452, -0.5452783107757568, 1.1362332105636597, -0.5530558824539185, -0.017589418217539787, 0.47920283675193787, 1.368556261062622, 0.43035435676574707, 0.2678924798965454, -0.060459502041339874, -0.24586763978004456, -0.36404988169670105, 1.1969963312149048, -0.47633591294288635, 0.10592833161354065, 0.20757760107517242, 1.3059122562408447, -0.9917850494384766, 0.5494522452354431, -0.5819776654243469, -1.8096716403961182, 0.6189996004104614, 0.5284722447395325, 0.35072827339172363, 0.3314247727394104, -1.365051031112671, -0.43272897601127625, 0.4122295677661896, -1.0761330127716064, 0.028129911050200462], [0.23653537034988403, 0.7283010482788086, 0.8738071322441101, 0.7412002086639404, -0.8559022545814514, 0.5637257695198059, -0.7275334000587463, -0.8818714022636414, -1.1118704080581665, -0.22953668236732483, -0.509384274482727, 0.3927295207977295, 0.4826316833496094, -0.9202490448951721, -0.478100061416626, 0.12526948750019073, 2.4607369899749756, -0.37227943539619446, 0.03468480333685875, -0.3348196744918823, 1.0865906476974487, -0.40586844086647034, -0.6494120359420776, -1.1983124017715454, 0.1739736646413803, -0.9316914081573486, 1.884523630142212, -0.7974557280540466, -2.017009735107422, 0.3693174123764038, -0.0030580146703869104, -0.4982241988182068, -0.36433058977127075, 0.5373812317848206, -0.11014299094676971, 0.23565219342708588, -0.30418872833251953, -1.4444735050201416, -0.054174844175577164, -1.319986343383789, -0.28809842467308044, 0.7141939401626587, 0.6023693084716797, -0.679370105266571, 1.9340952634811401, -0.765259861946106, 0.3791200816631317, 1.4179472923278809, 0.39993539452552795, 1.7692928314208984, -0.28363263607025146, 0.9546359181404114, -1.0017731189727783, -0.22647777199745178, -0.012531382031738758, -0.6593760848045349, 1.581411600112915, -0.5287492275238037, -0.009507300332188606, 0.997244656085968, -0.724496066570282, -1.343884825706482, -0.25370490550994873, 0.22059966623783112], [0.5897629261016846, -0.8412480354309082, 1.1277658939361572, 1.8617321252822876, 0.7792260646820068, -1.4134609699249268, 0.8928154110908508, 1.1141862869262695, 1.110730767250061, -0.8064955472946167, -0.9410516619682312, 1.3093929290771484, 0.016299568116664886, -0.3646433353424072, 0.7131126523017883, 1.202571153640747, 0.016997016966342926, 1.1971015930175781, -0.0034218912478536367, -2.366691827774048, 1.2910573482513428, 0.3855580687522888, -1.1074986457824707, -0.5488045811653137, 0.9598376750946045, 1.3919732570648193, -0.437016099691391, 0.5825802683830261, -1.0945106744766235, -1.0992285013198853, 0.5095837712287903, 0.38666266202926636, -1.3594919443130493, -0.6693292260169983, 1.8985402584075928, -0.7597602009773254, -0.8250373005867004, -1.4263060092926025, -0.8958310484886169, 0.7340903282165527, 1.2983494997024536, 0.8907177448272705, 0.3136325478553772, -0.5651844143867493, -1.1692670583724976, 0.47810935974121094, -1.343582034111023, -0.6224763989448547, -0.1802142709493637, -0.18485289812088013, 0.975342333316803, 1.9245482683181763, -0.20647995173931122, -0.9117180705070496, 1.4370440244674683, -2.4380650520324707, 0.857378363609314, 0.5812190771102905, 1.1639039516448975, 1.9976587295532227, 0.1265215426683426, 0.5743263363838196, -0.21163994073867798, -1.7197871208190918], [-0.8831207752227783, 0.07814759016036987, 0.7947152853012085, 2.01899790763855, -0.4226119816303253, -1.1188113689422607, -0.5996600985527039, -0.721225380897522, -0.32070639729499817, 1.0299460887908936, -0.039803389459848404, 0.07236286997795105, 1.3995294570922852, 0.6936458945274353, 0.8797782063484192, -0.4123347997665405, -2.830261468887329, -1.0481687784194946, -0.9222067594528198, 1.2833136320114136, 0.19391264021396637, 1.242468237876892, -0.3863137662410736, -0.35710135102272034, -0.6214414238929749, 1.037367582321167, 0.9290931224822998, 1.4089999198913574, 0.5868051648139954, -0.6436039805412292, 1.572229027748108, 0.5387409329414368, 0.3763120770454407, -1.0311245918273926, 0.4218113422393799, -0.18668979406356812, 0.00636956887319684, -0.014583677984774113, 1.478295922279358, 0.0649561882019043, -1.2598271369934082, 0.15874868631362915, -1.696847915649414, 0.39675718545913696, -0.4406384825706482, 0.18589988350868225, 0.3528558313846588, -1.4196089506149292, -0.4349585771560669, 0.4126957952976227, -0.6708574891090393, 0.29015395045280457, -1.2133647203445435, -0.6601267457008362, 0.5967612266540527, -1.3486342430114746, 1.707262396812439, 0.5160875916481018, -1.1570935249328613, 2.338590621948242, -0.9664677381515503, -0.3094903826713562, 1.624609351158142, 0.7161824703216553], [0.6295825242996216, -0.8287043571472168, -2.2277276515960693, 0.2300652116537094, -0.5693426728248596, -0.6367195248603821, 0.13379430770874023, 1.2957978248596191, 0.9335901141166687, 0.5702553987503052, -0.09084539115428925, -0.47464486956596375, -0.7379443645477295, -1.0704532861709595, 2.233163833618164, 0.32223597168922424, -1.181555151939392, -0.7285534739494324, -0.0049451133236289024, 0.5352922081947327, -0.051559239625930786, -0.02855653688311577, 0.3256329894065857, 1.5711421966552734, 0.929690957069397, -0.10381397604942322, -3.0790979862213135, 0.2940216064453125, 0.5212024450302124, 1.0129334926605225, -0.1650947779417038, -0.5500509738922119, -0.2311900109052658, -0.8207511305809021, 0.1085648164153099, -1.54381263256073, 0.9938656687736511, 0.8392771482467651, 0.9730817079544067, -0.6431873440742493, -1.6414527893066406, 1.8600088357925415, -0.2169416844844818, 0.18496465682983398, -0.6318532824516296, 0.46715840697288513, 0.29405370354652405, -0.23797601461410522, 0.061983879655599594, -0.3033120930194855, 2.6445953845977783, 1.4641567468643188, 0.9773227572441101, 1.797624945640564, 1.1184817552566528, -0.6327124834060669, -0.9598063826560974, -1.9183337688446045, 1.0445783138275146, -2.3602519035339355, 1.153964638710022, 0.25115060806274414, -1.8829237222671509, -1.8763549327850342], [-0.5137020349502563, -0.5062710642814636, -0.6209551095962524, -2.087785482406616, 1.016920566558838, -1.1211735010147095, 0.8106054067611694, 0.44666069746017456, 0.5549384355545044, 0.11699068546295166, -0.08755038678646088, -0.49722158908843994, -0.3998563587665558, -1.2145670652389526, -0.11297689378261566, 0.41127726435661316, 0.8086650371551514, 0.33534306287765503, 0.7278292775154114, 0.2500233054161072, 0.03703645244240761, 0.7659912705421448, 0.4606575071811676, -0.5758097171783447, -0.13278913497924805, -0.5565125346183777, -0.5694072246551514, -0.5016366839408875, -0.5840461254119873, -0.23036804795265198, 0.6685057282447815, -0.6992752552032471, -0.09676846861839294, 0.4959976077079773, -0.5215556025505066, 0.261407732963562, -0.1528429090976715, 0.8702016472816467, 0.002832916798070073, -0.4288204610347748, 0.5843992829322815, 0.062234651297330856, 0.3021426796913147, 0.2547982931137085, 0.7945805191993713, 0.31448429822921753, 0.8952749967575073, -0.780356228351593, -1.230720043182373, -1.0638811588287354, 0.6963144540786743, -0.9136419892311096, -0.28669795393943787, -0.8432568311691284, -0.5135005712509155, 0.46727511286735535, -0.8281552195549011, 0.6592183709144592, 1.4241474866867065, 0.2513014078140259, -1.0417107343673706, -0.7358831167221069, -1.023849368095398, 0.32339054346084595], [-1.5134166479110718, 0.29244670271873474, 0.6457411646842957, -1.172351360321045, 0.5469465255737305, 0.7263451814651489, -0.4948183298110962, -1.316199779510498, 0.04873348027467728, 0.8053829669952393, 0.30001604557037354, 0.014660888351500034, -0.35581228137016296, -0.2918216586112976, 0.6017858982086182, 0.39033791422843933, 1.6958316564559937, -0.617486298084259, -0.04912989214062691, 0.8118435740470886, 1.7068341970443726, 0.7333713173866272, -0.42231982946395874, -2.274547576904297, 0.9878002405166626, -0.46467071771621704, 1.1085115671157837, 1.24563467502594, -1.908345341682434, -0.8879365921020508, 0.8281405568122864, -0.03365492448210716, 0.8929365873336792, 1.0581977367401123, -1.2253482341766357, -0.7522870302200317, -0.6623941659927368, 0.11907148361206055, -1.1815979480743408, -0.6284643411636353, 1.332076907157898, 1.4425549507141113, 0.940363347530365, -1.3224706649780273, 0.03308174014091492, 0.32181623578071594, 0.8457292914390564, 0.4070184528827667, -1.6107633113861084, -0.4010809063911438, 1.2399314641952515, -1.7455580234527588, 0.94172203540802, 0.10343354940414429, -0.4889460802078247, -1.0109397172927856, -0.0525454580783844, 1.8153802156448364, -0.777090847492218, 0.08371887356042862, -0.1407550573348999, 0.9431614279747009, 1.2281980514526367, 0.1597832888364792], [-0.8006575107574463, -0.7662478089332581, 0.6044899821281433, 0.9543396830558777, 0.5572463870048523, -0.7572764158248901, -0.2854578495025635, -0.028334030881524086, -1.636291265487671, -0.7317619919776917, 0.39865562319755554, 0.61087566614151, 0.3304431438446045, 0.3172181248664856, 0.15680940449237823, 1.3569163084030151, -0.15776386857032776, 0.6504073143005371, -0.04913965240120888, -1.04340660572052, -0.879569411277771, 1.4855310916900635, -0.11369755864143372, -0.37953057885169983, 0.06424067169427872, 0.37286731600761414, -0.3308698236942291, 0.7491849064826965, 1.4551970958709717, 0.2694932520389557, -0.06706733256578445, -0.553881049156189, -1.0229549407958984, -0.7003297805786133, 1.052135944366455, 0.24161411821842194, 1.5722888708114624, -1.1870776414871216, 0.01797310635447502, 0.9668910503387451, 2.326850414276123, 0.9668306708335876, -2.3033292293548584, 1.2568730115890503, 0.9282524585723877, -1.359013557434082, -0.6871588230133057, 0.585369884967804, 0.6710180044174194, -2.0637850761413574, -1.1870981454849243, -1.0197194814682007, -0.09292268007993698, 0.31219881772994995, -0.4976865351200104, -2.1569173336029053, 0.4758269786834717, -0.3146198093891144, -0.9335903525352478, 0.06333915144205093, -0.750522255897522, -0.27765515446662903, -0.5483640432357788, -0.3092208802700043], [-2.118619680404663, 1.9182138442993164, 0.34762439131736755, -0.20126356184482574, 1.0957201719284058, -0.9906641840934753, -1.7674461603164673, 0.7600364685058594, 2.4866082668304443, -1.6258162260055542, 0.2924327254295349, -0.01958838477730751, 1.1299241781234741, -0.3534623980522156, -0.02773139253258705, -2.194730281829834, -0.7824851870536804, -0.6833060383796692, -0.2326986938714981, -4.173592567443848, 0.3091287612915039, 0.16312779486179352, 0.5403919816017151, 1.381055235862732, 0.07370113581418991, -0.5101941823959351, 0.932283878326416, 0.9227416515350342, 0.3908158242702484, 0.2755255103111267, 0.1492001861333847, -0.05894089862704277, -0.7333827614784241, -1.8472440242767334, 0.14744246006011963, 0.6131051778793335, -1.054221272468567, -2.519179344177246, 0.82845139503479, -1.361998438835144, -0.525934100151062, 0.2086838036775589, -0.4524673521518707, -0.45412400364875793, 0.6885927319526672, 2.0024633407592773, 0.12609140574932098, 0.436663955450058, -0.5396730899810791, 0.0911993607878685, -0.04317956790328026, 0.242209792137146, -0.8940812945365906, -2.285306930541992, 0.9885266423225403, 1.9052436351776123, -0.07041357457637787, 0.2072480320930481, 0.3246869742870331, 0.7416954040527344, -1.1693421602249146, -0.6576246619224548, 0.25924551486968994, -0.2676478326320648], [0.3827912509441376, 2.246522903442383, 1.810004472732544, 1.2801918983459473, 0.619408130645752, -1.558035135269165, 0.810609757900238, -2.6894164085388184, 0.8895958662033081, -0.5940154790878296, -0.38265204429626465, -0.6033968925476074, 0.5279447436332703, 1.5444597005844116, -0.03310512751340866, 0.7742612361907959, -0.09075475484132767, -0.575594961643219, -1.0917813777923584, 1.7704224586486816, 0.18079805374145508, 0.9855257868766785, -0.5734880566596985, -0.8376681804656982, 1.6731994152069092, 0.2038571536540985, 1.2332385778427124, 0.8566576838493347, 0.29088324308395386, 1.3090405464172363, -1.5451655387878418, -1.3994104862213135, -1.1133142709732056, 1.210776686668396, -0.053258609026670456, 1.5987576246261597, 0.13553428649902344, 0.5334900617599487, 0.2317655086517334, 1.6312469244003296, 0.1676701009273529, -0.42482826113700867, 0.9272637367248535, 0.48013779520988464, -0.17358899116516113, 1.5461703538894653, -2.149576187133789, 0.0800953283905983, -0.6054829955101013, 1.3535387516021729, -0.024027004837989807, 0.9513705968856812, -0.6535114645957947, -1.1639071702957153, 0.39008650183677673, -0.4819883108139038, 1.777397632598877, 1.3363077640533447, 0.02937682718038559, -1.892886996269226, -1.344197392463684, 1.3047096729278564, 0.3478444218635559, 1.1438531875610352], [-1.9815475940704346, -0.5483939051628113, -0.9863450527191162, -0.5339348316192627, -0.6027044057846069, -0.6460583806037903, 0.7066853046417236, -1.5023324489593506, -0.8839808106422424, -1.7766704559326172, 0.37250885367393494, 0.5859530568122864, -0.4978344142436981, 0.9535254240036011, -0.4204586446285248, -1.0857328176498413, -1.9423924684524536, -1.0873286724090576, -0.1201498806476593, -0.6720038056373596, 1.590506672859192, 1.4852468967437744, 1.2110607624053955, 1.1531920433044434, 0.7096381187438965, -1.4610018730163574, -0.9858177304267883, 0.2052587866783142, -0.8500547409057617, -1.0968363285064697, 0.6648465394973755, 0.2591986358165741, -1.4231549501419067, -0.8279829621315002, -0.42343276739120483, -0.15211157500743866, -0.8641253709793091, 0.29555264115333557, -1.2193018198013306, 1.019295334815979, -0.7027118802070618, 1.0046883821487427, 2.526397943496704, 0.6367732286453247, 0.8740420937538147, -0.8432745337486267, -1.1308292150497437, -0.41467225551605225, -0.05600805953145027, 0.5037776231765747, 0.2624621093273163, 0.16313612461090088, 0.32599660754203796, -0.9647980332374573, 1.226501703262329, 1.3772696256637573, 0.02765093743801117, -1.3187031745910645, 0.05475591868162155, -1.686186671257019, 0.3558138310909271, -1.2071782350540161, 1.036155343055725, -0.7858603596687317], [0.6993323564529419, -0.7184364199638367, 0.22920779883861542, 0.9842987656593323, 0.9181988835334778, 0.13169653713703156, 1.4006242752075195, 1.191903829574585, -0.39312583208084106, -0.37403181195259094, -0.7442573308944702, -0.8522000908851624, -0.6897575855255127, -1.6327238082885742, 0.6401456594467163, -0.404075026512146, 1.1433130502700806, -0.5104640126228333, 0.5082289576530457, -0.010411898605525494, -0.24520587921142578, 0.18918442726135254, -0.4435310661792755, -0.47417861223220825, -0.39219456911087036, 0.17028522491455078, -0.49095720052719116, 0.6276401877403259, -0.9742088317871094, -0.7522422075271606, -0.6638296246528625, 1.422903299331665, 1.1400169134140015, -1.2630436420440674, -0.3880675137042999, 0.1490871012210846, -0.4775582551956177, 0.6290788054466248, 0.6653987169265747, -0.018654359504580498, 1.8420137166976929, 0.5453553199768066, -0.8166582584381104, -0.35644376277923584, 0.05907781422138214, 0.14393584430217743, 1.7744348049163818, 1.993996262550354, -0.19455182552337646, 1.5825198888778687, 0.6505440473556519, 1.3612539768218994, 1.401212453842163, -1.7654510736465454, 0.1638137847185135, 0.7982539534568787, 0.15585404634475708, 1.7638306617736816, 1.3416951894760132, 0.9759302139282227, 1.3217647075653076, -1.151938796043396, -0.7055873870849609, -0.6171742081642151], [0.7216013073921204, 0.1999402940273285, -0.10747818648815155, 0.6794677376747131, 1.3851736783981323, 2.002728223800659, 0.14917320013046265, -0.6746214628219604, -0.6283408403396606, -0.8689748048782349, 0.32241684198379517, 0.9829537868499756, 0.39018523693084717, 0.3245193660259247, 1.0934340953826904, 1.0350301265716553, 0.745697557926178, -0.18289445340633392, -0.12178598344326019, -0.657193660736084, -0.41634026169776917, 1.072838306427002, -1.2952277660369873, 0.15052616596221924, 1.431592345237732, -0.9414511322975159, -0.22539709508419037, 0.43061086535453796, 0.04733886942267418, -1.1811366081237793, 0.07400240004062653, 0.210735023021698, -1.4383413791656494, 1.512012243270874, -1.5550014972686768, 1.623166561126709, -1.6278464794158936, 3.086452007293701, -0.5570169687271118, -0.9000873565673828, -0.2922186255455017, -0.22616176307201385, 1.2479528188705444, -0.6081879138946533, -0.24004943668842316, 1.3087929487228394, 2.105381965637207, -0.9944987297058105, 1.8711761236190796, 0.872522234916687, -0.0914251059293747, 0.4159555435180664, -0.2980158030986786, -0.7552154660224915, 0.8036798238754272, 0.6043153405189514, -1.6928625106811523, 1.0761637687683105, 1.5038129091262817, -2.055267333984375, 0.4771660268306732, -0.47516679763793945, -0.37605926394462585, -0.1529657244682312], [-0.3939756453037262, 1.237779974937439, 0.23940567672252655, 0.34832850098609924, 2.1429646015167236, -0.787635087966919, -1.2927802801132202, 0.7802447080612183, -0.19354954361915588, -1.2268555164337158, 1.9014954566955566, 0.3404789865016937, -1.2447351217269897, 0.4016388952732086, 0.45092788338661194, 0.3416498899459839, -0.5586645007133484, -1.513885259628296, 1.1291284561157227, 0.22118885815143585, -0.7259729504585266, -0.8493190407752991, -1.8078773021697998, -0.7869752049446106, -1.4772530794143677, 0.8876277208328247, 0.20775657892227173, -0.4349322021007538, 1.091030240058899, -1.6257623434066772, -0.32885944843292236, 0.02627325989305973, 1.0492897033691406, 2.0947794914245605, 0.5026312470436096, -1.4577109813690186, -0.9254056811332703, 1.28049898147583, 0.17114366590976715, 0.5382409691810608, -0.7160932421684265, 0.45281097292900085, -0.6490707397460938, -1.4547163248062134, -1.4558137655258179, -0.08166872709989548, -0.8386332988739014, 1.4337185621261597, 0.8970587849617004, -0.9172357320785522, 0.8826638460159302, 1.0483514070510864, -0.5889876484870911, 1.1910792589187622, -0.8558022379875183, -0.3320958614349365, 0.7442660331726074, -0.7926363945007324, 0.19597245752811432, 0.1347672939300537, 0.15853163599967957, -1.657454490661621, 0.6168801784515381, 2.1247692108154297], [-0.4414654076099396, 0.7197929620742798, -0.5325562357902527, -0.588675856590271, -0.3680635094642639, 0.6372190117835999, 0.598714292049408, 1.397226333618164, -1.4662485122680664, -0.7381106615066528, -0.45430323481559753, -0.5796840190887451, -0.044349491596221924, 1.300686240196228, 0.7166091799736023, -0.2823769152164459, 2.057874917984009, -1.2763837575912476, -2.4107413291931152, -1.4889849424362183, -1.1028302907943726, -1.5089234113693237, 0.3429737985134125, 1.2105566263198853, -0.08560843020677567, 0.029245516285300255, -0.7268573045730591, -1.1034103631973267, -2.115757942199707, -1.3499860763549805, 1.613318681716919, 0.3601725697517395, -0.5571045279502869, -1.224183440208435, -0.9500161409378052, 1.3269351720809937, -0.3550888001918793, -1.3963578939437866, -1.319536566734314, -0.7417111992835999, 0.3651019334793091, -0.020214367657899857, -0.5961004495620728, 0.1584663689136505, -1.0311992168426514, 0.6348879933357239, 0.40684714913368225, 1.217947006225586, -0.05371284484863281, 0.8385426998138428, -1.6763038635253906, 0.29965710639953613, -1.5520480871200562, -1.6045513153076172, 1.010140061378479, -0.9812825918197632, -0.9209884405136108, -1.34803307056427, 0.1438591331243515, 0.47676533460617065, -0.6704627871513367, 0.3495021462440491, 0.8711345791816711, 0.530767560005188], [-0.4687780737876892, -0.7188172340393066, -0.8862699270248413, -1.3693854808807373, -1.8951090574264526, 0.9821913838386536, 1.5296849012374878, 0.12206169962882996, 0.05160754173994064, 0.2949710190296173, 0.02675572969019413, 0.32586297392845154, 1.234542727470398, -0.6335659027099609, -1.3415225744247437, -0.7356339693069458, 0.5443019866943359, -0.022028610110282898, 0.4689961373806, 0.5813329219818115, -0.2955045700073242, 0.61128169298172, 0.33054396510124207, -0.012838619761168957, 0.587505042552948, -0.44736921787261963, -0.7378292083740234, 0.9412912726402283, 0.16960427165031433, 1.2938356399536133, -0.46018657088279724, -0.3814173638820648, 0.19927281141281128, 0.6950865983963013, -0.3589773178100586, 1.1043215990066528, -0.4763902723789215, 0.6786710619926453, -0.8681423664093018, 0.8445449471473694, 1.022826075553894, 0.9576871991157532, 0.4993598461151123, 0.364135205745697, 0.5359256267547607, 0.14381840825080872, 0.3261355757713318, 1.9289734363555908, -0.9992502331733704, 0.26032140851020813, 0.23799410462379456, 0.7957362532615662, 0.08737573027610779, -0.3063501715660095, 1.6399879455566406, -0.6025270819664001, -0.035779256373643875, 1.0499095916748047, 0.6303054094314575, -1.0129449367523193, -1.200717568397522, 1.6425635814666748, -0.8115617036819458, -0.5275654792785645], [-0.8220900297164917, 0.8183538913726807, 0.22978875041007996, -0.7235681414604187, 1.0197246074676514, -0.14886681735515594, -0.8846916556358337, -1.6356710195541382, 0.6149985194206238, -0.017606042325496674, 1.3052078485488892, -1.3870878219604492, 1.0952136516571045, 0.12240134179592133, -1.2648216485977173, 0.41894644498825073, 0.7427749633789062, -0.10224612802267075, -0.20032824575901031, -0.27676624059677124, 0.3970820903778076, -0.9010027647018433, 0.5750921368598938, -0.41398587822914124, -1.0017709732055664, -1.4111566543579102, -0.18447129428386688, 0.7387565970420837, 1.0027836561203003, 0.654886782169342, -0.9564259052276611, 1.8179576396942139, -0.6722398400306702, -1.1697869300842285, -1.17192542552948, -1.2109858989715576, -1.0246516466140747, 1.182483196258545, 2.5396292209625244, -0.23796017467975616, 1.6730284690856934, -1.866129994392395, 1.3260020017623901, 0.9156371355056763, 0.5713587999343872, -1.545077919960022, 0.6237591505050659, 1.0095782279968262, -0.32083413004875183, -0.9928221702575684, 2.5122950077056885, 1.652978777885437, -0.13464443385601044, -0.1387304663658142, 0.8652691841125488, 1.0261201858520508, -0.7651617527008057, -0.45748135447502136, -2.243867874145508, -0.3330936133861542, -0.20495182275772095, 1.9075971841812134, -0.024043021723628044, -0.9263211488723755], [-0.10892131179571152, -0.728867769241333, -0.8420951962471008, 2.193164110183716, -0.013651097193360329, -0.5954792499542236, 1.364437460899353, 0.7639878988265991, -0.6010711789131165, 0.26596394181251526, -0.5248141288757324, -0.1502317488193512, 1.8645753860473633, -0.048170056194067, -1.0076403617858887, -0.3547404706478119, 0.16640999913215637, 1.3488166332244873, -1.4169114828109741, 0.4413771629333496, 1.1387925148010254, -0.5977367162704468, -0.6545448303222656, 1.6490392684936523, 0.322355180978775, 1.7487170696258545, 0.8647308349609375, -0.16950970888137817, -0.6524648070335388, 0.9899399876594543, 0.1646198332309723, 0.33618590235710144, 0.10551221668720245, -0.47174355387687683, 0.25300323963165283, -0.03185637667775154, 0.6175511479377747, -2.858896493911743, -0.2783346176147461, 1.7362056970596313, -0.9355807900428772, 0.041946735233068466, -0.556001603603363, -1.4114532470703125, 1.1309845447540283, -0.24771644175052643, -0.6681510806083679, -0.15911051630973816, -1.038449764251709, 0.5262059569358826, 1.000718116760254, 0.6181240677833557, -0.9390225410461426, -0.020638540387153625, -1.8182768821716309, -0.6591052412986755, 0.33165010809898376, -0.8505384922027588, -0.30632519721984863, 0.5601517558097839, 1.1467458009719849, -1.2086524963378906, 0.7222428321838379, -1.162071943283081], [-0.8740266561508179, -0.6961635947227478, -0.8280102610588074, 0.20531415939331055, 2.46470046043396, 0.16685861349105835, -0.7841556072235107, 0.21616363525390625, 0.13362543284893036, 0.06790800392627716, 0.21163828670978546, 1.719275951385498, -0.34868958592414856, 1.6269152164459229, 0.4804770052433014, -0.6773719787597656, -2.2675893306732178, -1.6873139142990112, 1.0211375951766968, 0.9073320031166077, 1.5215625762939453, 0.5419695973396301, 0.6791420578956604, 0.2755137085914612, 2.0406508445739746, 0.12552152574062347, 0.3802527189254761, 1.351977825164795, -0.5646441578865051, 0.3644546568393707, 0.8862360715866089, 1.0135749578475952, 0.33093538880348206, 0.7144486904144287, 2.282489538192749, -0.5063498616218567, 0.893237829208374, -1.3000295162200928, 0.02192162536084652, 1.416339635848999, -0.4133559763431549, -0.8098419308662415, -0.5159845352172852, 0.14054395258426666, -0.03572560101747513, 0.3277004063129425, 0.6540092825889587, 2.630457639694214, 0.11389365792274475, 1.5507234334945679, -1.6218440532684326, -0.6741763353347778, 0.11770550906658173, 0.87838214635849, 0.001028840895742178, -1.0108749866485596, 1.9782847166061401, -1.319400429725647, -0.2956254482269287, 0.3710206151008606, 0.09135463833808899, -0.007303445599973202, 0.38157331943511963, -0.22182320058345795], [0.9153243899345398, -0.5333444476127625, -0.8072801232337952, -1.103747844696045, -0.5330204963684082, -0.05801016837358475, 0.4167611300945282, 1.412801742553711, -2.312107801437378, -0.6503241062164307, 1.3519375324249268, -1.4041014909744263, 0.07952974736690521, -0.3530728816986084, 0.7735710740089417, 0.9691016674041748, -0.47507166862487793, 0.27567943930625916, -1.2296569347381592, 0.5620034337043762, -0.22454920411109924, -0.5420495271682739, 1.628098487854004, -1.9372000694274902, 0.13733212649822235, -0.1531991958618164, -2.131474733352661, -0.2755134403705597, -0.8944531679153442, 0.25992247462272644, 0.6857256889343262, -0.6916177272796631, 0.8078301548957825, 0.121438167989254, -0.6181142330169678, -0.6145002841949463, -0.9697887897491455, -1.1905460357666016, -1.549177646636963, 0.8698185086250305, 1.2270302772521973, 1.3701504468917847, 1.2222142219543457, 0.015007225796580315, 0.3211822509765625, 1.800864577293396, 1.5130614042282104, 0.3610110580921173, -0.5294733047485352, -0.388240784406662, -1.3836872577667236, -0.3545539975166321, -0.42486265301704407, -1.4535032510757446, 0.6532047390937805, 0.6345910429954529, 1.3570656776428223, -1.8068969249725342, -1.941194772720337, -0.17089435458183289, -1.2309198379516602, 1.8916518688201904, 0.6240371465682983, -1.2593015432357788], [0.7755833268165588, -0.6062470078468323, -2.481412410736084, -0.9823516011238098, -0.07623385637998581, -1.5694938898086548, 0.5459861755371094, -0.9689729809761047, 0.8483901619911194, -0.670471727848053, -0.9406373500823975, 0.13033778965473175, -1.1644479036331177, -1.061907410621643, -1.977729082107544, -0.27790358662605286, -0.5112223029136658, -1.1300575733184814, -0.7486470341682434, -0.5068461298942566, 1.064357876777649, 0.9355956315994263, -0.09571140259504318, -0.8958923816680908, -0.1443697214126587, 0.138225719332695, -0.17945459485054016, 0.17772908508777618, -0.3802020847797394, 0.46429577469825745, -0.25537750124931335, 0.5534018874168396, -0.46331551671028137, 0.9484580159187317, 0.5689269304275513, 1.2739733457565308, 0.8342371582984924, -0.3457106947898865, -0.20003025233745575, -0.9196892380714417, 0.6230046153068542, 0.2713650166988373, -0.8625462651252747, 0.943864643573761, -1.125575065612793, -1.03484308719635, 0.014229662716388702, 1.07144296169281, 0.22388523817062378, -0.8272108435630798, 1.1630723476409912, 0.08185138553380966, 1.7741236686706543, -0.3961716294288635, -1.6712604761123657, -0.8319352269172668, -0.40367215871810913, 1.925989031791687, -0.33001670241355896, 1.7308955192565918, 2.0253429412841797, 1.5259475708007812, 0.25155165791511536, 1.7909784317016602], [-0.1947319060564041, 0.7614350318908691, 1.264538049697876, 0.8977730870246887, -1.3631410598754883, -0.44572439789772034, 1.6862894296646118, 0.07848189026117325, -2.067765712738037, 0.7875322103500366, 0.07408859580755234, 0.5548087358474731, 1.3625965118408203, 0.4969712495803833, 1.722344994544983, -0.7796823382377625, 1.5090813636779785, 1.6627895832061768, 0.2594415843486786, 0.405656635761261, -0.08040975779294968, -0.3187164068222046, 0.34188443422317505, 0.12743091583251953, -2.0632622241973877, -0.23788511753082275, -0.36923229694366455, 1.2939743995666504, -1.9315917491912842, 1.8183093070983887, 0.4816913604736328, -1.0523682832717896, -0.38103100657463074, 0.6488985419273376, 1.4796878099441528, 0.7816956639289856, -0.011468417942523956, 0.16934430599212646, 0.9328787922859192, -0.6729531288146973, -1.1036006212234497, -0.31683701276779175, -0.4111265540122986, -0.3145899176597595, -2.319779634475708, -1.4416636228561401, 0.007284851744771004, -1.370118260383606, -0.47754499316215515, 0.16553393006324768, -0.2098385989665985, -0.13851088285446167, 0.6977414488792419, 0.5430905222892761, 0.6663506031036377, -0.35486412048339844, 2.7703819274902344, -0.007143874187022448, 0.16481269896030426, 0.5215615630149841, 0.3234623968601227, 1.0728384256362915, 0.32133617997169495, -0.20755259692668915], [-0.24559007585048676, -0.4699891209602356, -2.1885952949523926, -0.514702320098877, 0.01352105475962162, -0.2826017737388611, 1.7062972784042358, 0.15483593940734863, 0.6729490756988525, -0.38074934482574463, -0.044747527688741684, 0.8795956373214722, 1.4900845289230347, -0.9510146975517273, 0.20139718055725098, 0.6997436881065369, 1.3421244621276855, -0.8450989127159119, -1.040631890296936, -2.233865261077881, -1.4769277572631836, 0.44395554065704346, -0.29744794964790344, 1.1243880987167358, 0.570635199546814, 0.5034464597702026, 0.7068844437599182, -1.5009753704071045, 0.821662187576294, -0.6831027269363403, -0.11554689705371857, -1.0432565212249756, 1.3358005285263062, 1.900912880897522, -0.15338578820228577, 1.3340611457824707, 1.5996357202529907, 0.42553579807281494, -0.1763533502817154, -0.41690874099731445, 0.06307126581668854, -0.6363570690155029, 0.7757802605628967, -2.0083889961242676, -0.5084845423698425, -0.4809033274650574, -0.867566704750061, -0.9122327566146851, -0.41916194558143616, 0.345663458108902, 0.8369024991989136, -0.3752748668193817, 0.5559191703796387, -1.0588771104812622, -0.6675083041191101, -0.07592637091875076, -0.037813249975442886, -0.12916439771652222, -0.4533497095108032, 0.6686063408851624, -0.6572843790054321, -1.3340269327163696, -0.9681439399719238, 2.220506429672241], [-0.9928824305534363, -0.6891599893569946, 0.6786903738975525, -0.68623286485672, 2.185654878616333, -0.3092334568500519, -0.7005869746208191, -0.7152242660522461, 1.1749820709228516, 0.48738232254981995, 1.6255853176116943, 1.7462053298950195, -1.0711675882339478, -0.39490437507629395, 1.8451451063156128, -0.49678725004196167, -0.5994594693183899, -1.9577524662017822, 0.48002558946609497, 0.5905811786651611, 1.3799127340316772, 0.30538409948349, -1.3764218091964722, 0.10569221526384354, -0.021676065400242805, -1.1534866094589233, -0.334708034992218, 0.8779757618904114, 1.0748891830444336, 0.9227868318557739, -0.28076839447021484, -0.015148136764764786, 0.8062618970870972, -0.18808382749557495, -0.5323379039764404, -0.4827684462070465, 0.17410996556282043, 0.7881086468696594, 0.07640991359949112, -0.7868699431419373, 0.3708042800426483, 1.4235625267028809, -0.45732825994491577, -0.9614577889442444, 0.17569975554943085, 0.9954599142074585, -1.0443801879882812, -0.402256578207016, -0.38617873191833496, 1.471732497215271, -0.9883118271827698, -0.471832811832428, 0.6261526346206665, -0.7992194294929504, -1.892233967781067, 0.4932909309864044, -0.6390876173973083, 3.3207509517669678, 2.0215647220611572, -0.6438468098640442, 0.5865570306777954, 1.1162548065185547, 0.981889545917511, 0.6802590489387512], [0.24244149029254913, -1.0185272693634033, 1.4675484895706177, 0.5321070551872253, 0.030356531962752342, 0.04660196602344513, 0.17021708190441132, -0.20164817571640015, -1.1400947570800781, 0.7011706233024597, 0.45137128233909607, -0.7347983717918396, -0.7173944711685181, 0.81731778383255, 0.5553110837936401, -1.2590751647949219, 1.0743155479431152, -0.43022260069847107, 0.36741942167282104, 1.5067123174667358, 0.7475147843360901, 1.2002164125442505, -0.8176316618919373, 0.8321352601051331, -1.499252438545227, -0.7570212483406067, 1.2694032192230225, 0.47793978452682495, 0.8780632615089417, -1.0391151905059814, 1.5540030002593994, -0.5879147052764893, -0.07024788856506348, -0.8845077157020569, -0.03432954475283623, 0.30815890431404114, 0.3408043682575226, -1.7922698259353638, -0.6200250387191772, -0.7697033286094666, -0.008670832961797714, -0.21806687116622925, -2.071333646774292, -0.8029491305351257, -0.6554199457168579, 0.1890978217124939, 0.24315610527992249, 0.3044549524784088, -0.8749095797538757, 1.7702946662902832, 1.3667138814926147, -0.24216611683368683, -0.3948153853416443, -2.6953203678131104, 0.14445893466472626, -0.14575430750846863, 0.71190345287323, -0.02923109009861946, -1.7413740158081055, 0.6617841720581055, -0.461790531873703, -1.8450217247009277, 0.0567963607609272, 0.33083534240722656], [1.619313359260559, 1.1391723155975342, 0.5952842831611633, -1.1598305702209473, 0.052466586232185364, -0.3056133985519409, -0.3562450110912323, 0.9588534235954285, -0.9224282503128052, 0.6683003902435303, -0.18552200496196747, 0.3154352903366089, -0.09195324778556824, -0.48389932513237, -1.765081763267517, -0.09661105275154114, 0.31454187631607056, 1.071418285369873, -0.6538658142089844, 0.17624087631702423, 1.0221258401870728, -0.55143141746521, 0.30418720841407776, -0.06017312407493591, 2.0816876888275146, 1.7551804780960083, 1.1836220026016235, 0.31158027052879333, 1.078842043876648, 1.0084317922592163, -0.5270078182220459, 2.6206915378570557, 1.137549638748169, 0.8369561433792114, 1.0969120264053345, -0.8590541481971741, 1.278058648109436, 0.48719143867492676, -1.5381110906600952, 1.1410585641860962, -0.6192428469657898, -0.9930092692375183, -0.38715827465057373, -0.23502153158187866, 0.27197661995887756, 0.743137001991272, 0.6547120213508606, -0.42377471923828125, -0.07712224870920181, -1.736770749092102, -1.0656380653381348, -0.9248090982437134, -0.8430081009864807, 0.16184991598129272, 0.7663639783859253, 0.27992957830429077, 2.297593116760254, -0.28987428545951843, 0.2552575469017029, 1.0808273553848267, 0.9731768369674683, -1.6617804765701294, -0.3845216929912567, 0.2925838232040405], [-0.6634893417358398, 1.0122112035751343, -1.1198245286941528, -1.6121984720230103, 0.768928587436676, -0.6441816091537476, 0.5777105093002319, -0.2857382595539093, 2.021883010864258, 1.9781404733657837, 0.38020601868629456, -2.114885091781616, -0.3111943304538727, -1.285498023033142, -0.0672597885131836, 0.9256752729415894, -0.8711869120597839, 0.10084956139326096, 0.24884812533855438, -0.8038840889930725, -0.5136901140213013, 1.9503058195114136, 0.09552638232707977, -0.6486780643463135, 0.1915096491575241, -0.24966250360012054, 0.17040662467479706, 1.1652684211730957, -0.021539313718676567, 0.9160153865814209, -0.35526320338249207, -0.20989422500133514, 0.282527357339859, -0.2784888446331024, -0.07473341375589371, -1.2522228956222534, 1.5260231494903564, -0.2522967457771301, -0.29196569323539734, -0.8912578225135803, -0.3859136402606964, 1.0502064228057861, 1.0066099166870117, 0.5280989408493042, 1.7729212045669556, 0.41867977380752563, 0.40699049830436707, 0.8521515130996704, 0.054573897272348404, -0.08340365439653397, -2.9800283908843994, -0.5176829099655151, -1.4995514154434204, -1.0026229619979858, -1.516342282295227, -0.46916115283966064, -0.13466478884220123, 0.41795629262924194, 0.6535637378692627, 0.47439315915107727, 0.43590912222862244, -0.5044329762458801, -0.7100715041160583, 1.8886488676071167], [0.35121950507164, -0.3304837644100189, -1.6874083280563354, -0.24603977799415588, -1.6846812963485718, 0.9031263589859009, 0.10040171444416046, -1.5656620264053345, 0.6569387316703796, 1.8409335613250732, 1.0461210012435913, 0.08158084005117416, -0.5774837732315063, -0.2320314645767212, -0.992241621017456, -0.6681210994720459, -0.2308974713087082, 0.23584075272083282, 0.7628681063652039, 0.48848867416381836, 1.1232104301452637, -0.6054783463478088, -0.6418241262435913, -0.5365405082702637, -0.8005347847938538, -0.2887812852859497, -0.4931844174861908, -1.1104305982589722, -0.5859618782997131, -0.23402781784534454, 0.5445876717567444, -0.045040249824523926, 1.7661083936691284, 1.6410773992538452, -2.952106475830078, -0.16560526192188263, -1.1709105968475342, 0.4761711657047272, 0.3081458508968353, -0.14271284639835358, -0.5330275893211365, -0.06849853694438934, -1.3753135204315186, 0.3050108850002289, 0.5270135402679443, -0.17464125156402588, -0.2580132484436035, -0.8549674153327942, 1.3008875846862793, -0.7128929495811462, -0.5582497119903564, -1.763395071029663, -1.3201119899749756, -0.42308518290519714, 0.5908441543579102, 0.05433272570371628, 0.019221875816583633, -0.462816059589386, -0.7601451873779297, 0.7938624620437622, 0.39212876558303833, -1.1110508441925049, -0.8206092119216919, 0.07356078922748566], [-0.5218307971954346, 1.6339869499206543, -0.9813430309295654, -0.547572910785675, -0.09667975455522537, 0.530471682548523, -0.6583545207977295, 0.45392829179763794, 0.33767202496528625, 1.06108558177948, 0.8100318312644958, 0.8717997074127197, -0.2971978783607483, -2.8624541759490967, 0.6237362027168274, 0.027074549347162247, -0.6304768919944763, -2.0171594619750977, 0.6417264342308044, -2.8821749687194824, 1.5024844408035278, -0.1087324395775795, -0.18033072352409363, -0.33154359459877014, 0.966148316860199, 0.11227057129144669, 0.6379018425941467, 0.5347154140472412, -0.33846890926361084, -0.15796327590942383, -1.4805132150650024, -1.3453682661056519, -1.4674874544143677, 1.7437670230865479, 0.6216432452201843, -0.33078864216804504, 1.141653299331665, -0.20722231268882751, -0.32164886593818665, 0.5150411128997803, 1.4112576246261597, -0.23803962767124176, 0.952039361000061, -1.0078296661376953, 1.4000976085662842, -1.774158000946045, 0.7293906211853027, 0.07658103108406067, -0.12060767412185669, -0.687188446521759, -1.7817225456237793, 0.4657270014286041, -0.7728420495986938, 0.6802855134010315, 0.8125154376029968, -0.5587160587310791, 0.47605231404304504, 0.4999516010284424, -0.5800888538360596, -1.7551004886627197, 1.6035808324813843, 1.064305305480957, 0.01613253727555275, 0.0315418541431427], [-0.4981672465801239, -1.3378806114196777, 0.2901642620563507, -1.976950764656067, 0.48326748609542847, 0.06161101162433624, 0.1996501386165619, 1.3653440475463867, 0.7848146557807922, 1.5643757581710815, 0.40464287996292114, -0.4286865293979645, -0.2950493097305298, -0.9840145707130432, 1.6576037406921387, -1.0123671293258667, 1.3816068172454834, 0.7251704931259155, -2.3857314586639404, 0.6799514889717102, 0.5971154570579529, -0.5435394048690796, 0.027776524424552917, 2.080493211746216, 0.04012787714600563, 0.5363513231277466, 0.6321287751197815, -1.5488040447235107, -1.7793467044830322, 0.469956636428833, -0.6909787058830261, 2.665105104446411, -0.6950904726982117, -0.4589254856109619, 0.7745378017425537, -1.0714861154556274, 0.9590446352958679, 0.7530657052993774, 0.765318751335144, -0.38420602679252625, -2.425842761993408, -0.36785972118377686, 0.7477923035621643, 0.5154356360435486, 0.52713543176651, -2.731560707092285, 2.326136827468872, 0.7680254578590393, 0.5145236849784851, 0.8767026662826538, 0.050170183181762695, 0.9380013942718506, -0.47761961817741394, -0.47438180446624756, -1.0845295190811157, -0.12613919377326965, -0.036193542182445526, -0.9777476787567139, 1.5165811777114868, -1.380661964416504, 1.2288963794708252, -2.9680511951446533, 0.031166942790150642, -0.41995516419410706], [0.9841075539588928, -1.152997612953186, 1.2311137914657593, -0.3139161467552185, -1.1947036981582642, 1.3761987686157227, -2.1455321311950684, 0.343682199716568, -0.005031641107052565, 0.217177614569664, -2.2747159004211426, -0.09000808000564575, 1.4607375860214233, 0.4049721658229828, 0.5052038431167603, 1.2361541986465454, -1.535968542098999, 1.280879259109497, -0.6362078785896301, -1.3088626861572266, 0.3850793242454529, -0.5413220524787903, -0.1845516562461853, -2.2547831535339355, 0.20674186944961548, -1.0559059381484985, 1.4963629245758057, -2.0185863971710205, -0.8255689740180969, 0.8764697909355164, -0.13629916310310364, 0.7072367668151855, 0.8022177219390869, 0.33627375960350037, 0.40460631251335144, 0.010698722675442696, -1.1412097215652466, 0.6658244132995605, 0.17357157170772552, -1.5020462274551392, 0.025264611467719078, -0.025930102914571762, -0.7717965841293335, -1.1446316242218018, -2.2484633922576904, -2.350881814956665, 2.1411800384521484, 0.7694991230964661, -0.13466770946979523, -2.2140910625457764, -0.33694586157798767, 0.9099137783050537, -0.8723177313804626, -0.7545735836029053, -0.06533519923686981, 0.8899473547935486, -0.14400510489940643, -1.1562671661376953, 0.7170127630233765, 0.40260642766952515, 1.1102432012557983, 0.06738633662462234, 0.8585018515586853, 1.4142634868621826], [-0.2503957450389862, 0.2263941615819931, -0.5720572471618652, 0.791545033454895, 0.6412147283554077, -0.9764518141746521, -0.28472140431404114, 1.3137298822402954, 0.7779140472412109, 0.36634334921836853, -0.7991206049919128, -0.21116073429584503, -0.5193263292312622, -0.6063981056213379, -1.3048962354660034, -1.4140440225601196, -1.56642746925354, 0.858429491519928, -1.3276965618133545, 0.71859210729599, 1.7862571477890015, -0.1486564725637436, -1.0834347009658813, -1.0591753721237183, 1.616276741027832, 0.23450621962547302, -0.605855405330658, -0.20837493240833282, -0.08307668566703796, -2.4014041423797607, 0.9544397592544556, 0.052796971052885056, -0.2875742018222809, -0.5505743026733398, 0.9656582474708557, -0.31879696249961853, -0.5850815176963806, 0.15246854722499847, 0.8406772017478943, 0.5929118990898132, 0.46754157543182373, -0.022181423380970955, 0.6448497176170349, 0.0959879457950592, -0.4666135609149933, -0.4835376441478729, 1.3177964687347412, 1.555682897567749, -0.04744724929332733, 0.4023793935775757, 0.6678407788276672, 2.648362159729004, -1.0041931867599487, -0.001835737726651132, -0.29998520016670227, -0.49772828817367554, 1.193257451057434, -0.08087871968746185, -0.23708555102348328, -0.8403880596160889, 0.945233941078186, 1.4715701341629028, 1.3963578939437866, 0.08867491036653519], [-0.7557635307312012, -0.17052079737186432, 0.5621811151504517, -0.30368930101394653, -1.0680040121078491, -0.1669415980577469, 0.6042034029960632, 0.292591392993927, 0.6669118404388428, -0.17036262154579163, 0.607609212398529, 0.21207500994205475, 0.7605052590370178, -0.2097255438566208, 2.2363009452819824, -0.5397114157676697, -0.4915158450603485, 0.32394668459892273, 1.480268955230713, -0.5579919219017029, 0.6074908375740051, -1.490769386291504, 0.5563536286354065, 1.501763939857483, 1.6846293210983276, 0.1709333062171936, 0.7352027297019958, 0.17353104054927826, 1.0245749950408936, 0.24834264814853668, -0.2328638881444931, 0.6850038766860962, -0.8362151384353638, -0.390890508890152, -1.4262264966964722, -0.6234840750694275, 0.33256104588508606, -0.29149091243743896, -0.5769259929656982, 1.4729969501495361, -1.1986684799194336, -3.404050350189209, -0.3123246729373932, 0.26565876603126526, 1.1653823852539062, -0.5526430606842041, -0.16594183444976807, -0.18798869848251343, 1.7675728797912598, 0.21781620383262634, -0.22972871363162994, -0.742808997631073, -0.3354796767234802, 0.46778860688209534, 0.30015647411346436, 0.9927365779876709, 0.75203937292099, 0.46038541197776794, 0.7437199950218201, 1.187198519706726, 0.5017152428627014, 2.5431344509124756, -0.3718532919883728, -0.1710534244775772], [1.373565673828125, 1.1914523839950562, -0.6049509048461914, -0.25334733724594116, 0.027795279398560524, 0.7662318348884583, -0.669433057308197, 3.0931620597839355, -1.197272777557373, -0.3612714111804962, 0.8876866698265076, -0.19820275902748108, 1.1250030994415283, 0.6737200021743774, -3.4142794609069824, -1.0317052602767944, -0.5509098768234253, -1.1755082607269287, -0.7905797362327576, 0.8164345622062683, 0.7291409969329834, -0.8307321071624756, 0.13236264884471893, -1.9518367052078247, -1.804408073425293, -1.287401556968689, -2.3024039268493652, -0.8889731168746948, -0.8715574741363525, -1.1862894296646118, 1.0813099145889282, -1.1395413875579834, -0.247177392244339, -0.14694063365459442, 0.48786890506744385, 0.48210448026657104, -0.08468634635210037, 0.1008836105465889, 1.9007116556167603, -0.14855901896953583, 0.4683637022972107, 0.4194924831390381, 0.3833218216896057, 1.5922123193740845, 1.4676381349563599, 1.0282912254333496, -1.1164569854736328, -2.6009576320648193, 0.482619971036911, 1.1899466514587402, 0.406938761472702, -1.1068463325500488, 0.07838734984397888, -1.7203933000564575, -0.09954749047756195, 1.7459290027618408, -0.05009002611041069, -0.2277996987104416, 0.4800884425640106, 0.47627151012420654, -0.3162582814693451, -0.17843814194202423, -0.3967498242855072, -1.2448627948760986], [0.21040570735931396, -0.3657110631465912, -1.092846155166626, 2.070974826812744, -0.26581162214279175, 0.028165876865386963, 0.9789084196090698, -0.25492608547210693, 0.31448930501937866, 1.345002293586731, -0.24197795987129211, 0.7853993773460388, -0.7573837637901306, -0.026835957542061806, -1.0277276039123535, -0.9732878804206848, 1.0449624061584473, -0.109907366335392, -0.26592734456062317, -0.2454226315021515, -1.0387656688690186, -0.4605971872806549, 0.31469714641571045, 0.7478041052818298, -1.4688714742660522, -0.41178569197654724, 0.6007550954818726, -0.821689784526825, 0.4830147922039032, -0.6276321411132812, 0.4342168867588043, 1.2650700807571411, 0.09794063121080399, 0.008201081305742264, -1.0637937784194946, 1.2120007276535034, -0.9344197511672974, -0.7436107397079468, -0.590043306350708, 0.7273350358009338, -0.9284941554069519, -0.2671715021133423, -0.2910362780094147, 0.14868178963661194, -0.007257436402142048, 0.31658467650413513, -1.5212335586547852, 1.604438066482544, -1.3294928073883057, -0.8606293797492981, -0.27302849292755127, 0.45689231157302856, -2.252227306365967, -1.8222728967666626, 0.3208482563495636, -1.0770522356033325, -0.5184763669967651, -2.040937900543213, 0.525046169757843, -0.5327850580215454, -0.5734694600105286, 0.10317966341972351, -0.0616416297852993, 1.071911334991455], [2.258091449737549, -0.05038846284151077, 0.12191243469715118, 0.5411158800125122, -0.5612763166427612, 0.779930055141449, -1.9799870252609253, -0.07483725994825363, 0.5838834643363953, -1.8363884687423706, -0.012345731258392334, -2.05084490776062, -1.0122727155685425, -0.6708895564079285, -0.17391420900821686, 0.8996662497520447, 1.4676746129989624, 0.28723782300949097, -1.7597763538360596, 0.22126279771327972, -0.24295073747634888, 0.04260232672095299, -0.11782475560903549, 0.018805447965860367, 0.9357489347457886, -0.7476208209991455, -0.20449818670749664, -1.2123788595199585, 0.029029790312051773, 2.470022201538086, 0.29109665751457214, -0.047555990517139435, 0.08934328705072403, -0.2238253951072693, 0.06861955672502518, 0.40649840235710144, -2.9503462314605713, -0.6680535078048706, 0.4096074104309082, 0.2841074764728546, -0.07471482455730438, 0.7911370396614075, -1.6497396230697632, -1.6605573892593384, 1.2796308994293213, 0.4595819413661957, 1.6858627796173096, -0.44462713599205017, 0.49142464995384216, -1.284318208694458, -1.2461069822311401, -0.30452802777290344, -1.6328378915786743, 0.3950411379337311, -1.4025986194610596, -0.6090508699417114, -0.6121119260787964, 0.9785943031311035, 0.9558836221694946, 0.0026032361201941967, -3.0222060680389404, -1.0664591789245605, -0.44757720828056335, 1.0883594751358032], [-0.10591831803321838, -1.4762202501296997, 2.461725950241089, 1.3006319999694824, 0.2554055154323578, -0.18925805389881134, -0.9159700274467468, 0.34210339188575745, 0.12524402141571045, 0.08017709851264954, 0.4027312397956848, 1.1680622100830078, -1.4320248365402222, 0.13094379007816315, -0.6944167017936707, -1.0154240131378174, -0.796903669834137, 0.3814019560813904, -0.5680155158042908, 0.36168867349624634, -0.5584899187088013, 1.233141541481018, 0.6571980714797974, -1.4120601415634155, 0.6380168795585632, -0.9823039174079895, 1.164191722869873, -0.7775384783744812, 1.3810551166534424, 0.714671790599823, -1.3163771629333496, -0.25807586312294006, -0.24335405230522156, -0.22542573511600494, -0.7225921750068665, -0.36315372586250305, 0.14648102223873138, 0.053685545921325684, 1.0335209369659424, -1.5789740085601807, 0.5626927018165588, 0.6869565844535828, 0.2540808618068695, 0.36331143975257874, 0.7366817593574524, -0.34546181559562683, -0.8609285950660706, 0.196044459939003, 1.3077483177185059, -0.23328600823879242, -2.412058115005493, 0.6443586945533752, 1.5033259391784668, -2.2803547382354736, 0.2800368368625641, -1.0419445037841797, -0.04650941863656044, -0.2786346673965454, 1.013913869857788, 0.03492755442857742, 1.801533579826355, 0.5002344846725464, 0.3983432948589325, -0.9749287962913513], [1.991282343864441, -1.5742849111557007, -0.29371142387390137, 1.3976049423217773, -0.7626757621765137, -0.09405318647623062, 0.15582318603992462, 0.7999076843261719, -1.3042526245117188, -0.8936975598335266, 0.4220517575740814, 2.47100567817688, 1.9964323043823242, 0.7050662040710449, 0.6520118713378906, -0.43517401814460754, -1.701482892036438, 0.5804486870765686, 0.058671627193689346, -1.2151929140090942, -0.6568718552589417, -0.7428741455078125, 0.2943289875984192, -0.49628883600234985, 0.674938440322876, 0.9624871611595154, -0.7715824246406555, -2.045316457748413, 0.7073976993560791, -0.3439040184020996, 0.6236357092857361, 0.6466013789176941, -1.06185781955719, 0.9632973074913025, 0.12981250882148743, -2.4115447998046875, -0.6434267163276672, -1.439100980758667, -1.4334080219268799, -0.7767451405525208, -0.8762243390083313, 0.9004781246185303, 0.6859288215637207, -1.8223912715911865, -0.48097607493400574, 1.2810027599334717, 0.7682592868804932, -0.14101961255073547, -0.7511702179908752, -0.4882706105709076, 0.8573018312454224, 0.18092069029808044, -1.6964199542999268, -0.005472416058182716, 1.5518020391464233, 0.22975532710552216, -1.2329717874526978, 0.00958687998354435, 0.766136109828949, 0.23432433605194092, -0.2261165976524353, -0.0496891550719738, -1.4943046569824219, 1.4401687383651733], [-0.45595917105674744, 0.8024995923042297, 2.2846357822418213, 2.045755624771118, -0.031907204538583755, -1.4508767127990723, -1.106742024421692, 0.3771968185901642, -1.1292484998703003, 0.10216914117336273, -0.1833488792181015, -0.22703434526920319, -2.4738171100616455, 0.0630827248096466, 0.38159698247909546, 0.1250394582748413, -1.6318299770355225, -0.7867119908332825, -0.5734386444091797, 0.9810344576835632, 0.47066861391067505, -0.9892037510871887, 1.667939305305481, 1.0242218971252441, 0.4399348497390747, -0.8328814506530762, 1.836250901222229, -0.8450562953948975, -0.15555155277252197, 1.71481192111969, -1.242369294166565, 0.6803285479545593, -0.9916797280311584, -2.1777102947235107, 1.2046653032302856, -1.531512975692749, -0.5863725543022156, -0.840045154094696, 0.7362127900123596, 0.32807451486587524, -0.10628920793533325, -0.13733011484146118, 1.3233113288879395, -0.07303116470575333, -0.35722002387046814, -0.6512109041213989, 0.03325652703642845, 1.316226840019226, -0.08791099488735199, -0.6609414219856262, -1.0316790342330933, 2.6104366779327393, 0.7978630661964417, -1.79445219039917, 0.047288279980421066, -1.3490136861801147, -1.043540596961975, -1.0414764881134033, 0.02065732330083847, -0.40738576650619507, -0.9545767903327942, -0.3615182638168335, -0.5647063851356506, -0.528628408908844], [-1.529945969581604, 0.3293820023536682, 1.3613489866256714, 0.6167101263999939, 0.16386468708515167, -0.169615238904953, -0.1304764747619629, -0.6638336777687073, 3.0765068531036377, 1.9074146747589111, -1.3286856412887573, -0.7318001985549927, 0.6643913388252258, -0.7093327641487122, 0.27968302369117737, -1.9807782173156738, 0.9496119618415833, 0.10267630219459534, -1.8186205625534058, -0.541668176651001, -1.0723987817764282, 0.9340075850486755, 0.43062591552734375, 0.9679000973701477, 0.37665265798568726, 0.4796709418296814, -0.9171256422996521, 0.26700159907341003, 1.700179100036621, 0.622201681137085, -0.09543521702289581, -1.3034473657608032, -0.3488021790981293, 1.543256402015686, 0.8695865869522095, -0.19464543461799622, 0.11774929612874985, 0.3215060532093048, 2.991527557373047, -0.4566875696182251, -0.39907559752464294, -0.7497182488441467, 2.6805171966552734, -0.7471714615821838, -1.0165890455245972, -0.5714216828346252, 0.713019073009491, 0.6441278457641602, -1.5551073551177979, 0.48461827635765076, -2.1278672218322754, -0.7385063767433167, -0.9582126140594482, 0.7399790287017822, -0.4930960536003113, 1.1566314697265625, 0.8623337149620056, -0.9321713447570801, -0.4195403456687927, -0.34334486722946167, -0.1853494495153427, 1.1600782871246338, -0.7040445804595947, 1.3377659320831299], [-1.2132258415222168, -0.8064228892326355, -0.04182391241192818, 0.3685847520828247, -1.6412394046783447, -1.1250394582748413, -0.9328561425209045, -0.03911178559064865, 0.8618976473808289, -0.09273785352706909, 0.36161676049232483, -0.06187783181667328, 0.801343560218811, 0.5298941135406494, 0.263813853263855, 1.1047922372817993, -0.4158158004283905, -1.621923804283142, -0.6105791926383972, 0.7255929708480835, -0.8054730296134949, -0.8495908975601196, 0.17229831218719482, -0.43249449133872986, 0.5755671858787537, 0.5495864748954773, 0.5830069780349731, -0.023365871980786324, -0.6066029071807861, -0.8322338461875916, 0.421811044216156, -0.35407939553260803, 0.3759048581123352, -0.12216956913471222, -0.10513453930616379, 1.2025643587112427, 1.2450532913208008, -2.0299439430236816, 1.7843843698501587, 1.1859064102172852, -1.8573240041732788, 0.6440199017524719, -0.0006603127112612128, 1.7628625631332397, 0.5828551650047302, -1.1137611865997314, -0.7141162157058716, -0.2686624526977539, -0.29318469762802124, -0.2903129756450653, 0.5323594212532043, 0.3564564287662506, 0.821165919303894, -0.31041431427001953, -0.27573931217193604, -1.1658141613006592, -1.3398131132125854, -0.055814001709222794, -1.1002784967422485, -0.1392921358346939, -0.6067911982536316, -0.9400320053100586, -1.0805109739303589, 0.4675136208534241], [-1.7850803136825562, -0.84644615650177, -1.269816517829895, 1.3359829187393188, -0.24351978302001953, 1.7216030359268188, 0.25889846682548523, 0.5630946755409241, 0.1329880654811859, -0.07283986359834671, -0.31449010968208313, -1.66774582862854, -0.38507401943206787, -0.04337140545248985, 0.3241562843322754, -0.8614607453346252, -0.18773818016052246, 0.40872830152511597, -0.6015328764915466, -0.36012256145477295, 0.5584116578102112, 0.11281614005565643, -0.5945465564727783, 0.9614521861076355, 0.9055801630020142, 0.49692070484161377, -1.3316314220428467, -0.8619153499603271, 0.7243484258651733, 0.9534650444984436, 0.836991012096405, -0.16896076500415802, 0.45885011553764343, 0.5292178392410278, -0.5549106001853943, -1.9876104593276978, -0.4448000490665436, -1.5636518001556396, -0.23351427912712097, 1.2480705976486206, -0.3767760097980499, 1.0265074968338013, -0.23100680112838745, 1.3702635765075684, 1.1391348838806152, -0.9547110795974731, 1.0618717670440674, -0.3669452369213104, -0.7480133175849915, -2.7109923362731934, -0.24992626905441284, -1.247718095779419, 0.04959315061569214, 1.5164880752563477, -0.26526662707328796, 1.2324700355529785, -0.08370460569858551, -0.6375190019607544, 2.5027267932891846, 0.9677938222885132, -0.8333548307418823, 3.550546407699585, 0.8441054224967957, 0.5806971192359924], [0.3392886817455292, -1.775816559791565, -0.5451241731643677, 1.2603061199188232, -1.3564778566360474, -0.05588415265083313, -1.2258200645446777, 0.8329992294311523, -1.1335104703903198, 0.24783454835414886, -0.40219080448150635, -1.0034658908843994, -1.0475901365280151, -1.8515663146972656, -0.7543104887008667, -1.1905016899108887, 0.26037082076072693, -0.7490635514259338, 1.0360770225524902, -0.31360924243927, 0.21237391233444214, -0.09693896770477295, 0.5509229898452759, 0.3765905201435089, 0.36673107743263245, -0.775913417339325, -0.6997395753860474, 1.121759295463562, 1.001112461090088, 1.2653712034225464, -0.40403589606285095, 0.5253594517707825, -0.48492100834846497, -1.5534718036651611, -1.281507968902588, 0.26854178309440613, 1.9564884901046753, 0.9842896461486816, -0.5973908305168152, -0.41786980628967285, -0.42786678671836853, -0.35368263721466064, 1.5055434703826904, 0.49999260902404785, -0.3247561752796173, -0.03587882220745087, -0.6133776307106018, 0.1883469969034195, -1.0988280773162842, -1.268991470336914, 0.5810316205024719, -0.8448135256767273, 1.0804734230041504, 0.6380140781402588, 0.052123598754405975, 1.686905860900879, -0.06750009208917618, 0.3372749984264374, 0.6644724607467651, -0.6403853297233582, 1.2306350469589233, 1.0911186933517456, 0.7602282166481018, -1.2524383068084717], [0.4566885530948639, -0.6396811008453369, -0.45390021800994873, -1.3414939641952515, -1.1419075727462769, 0.02336645871400833, -0.17950725555419922, -1.3472251892089844, -1.304157018661499, 0.37855497002601624, -0.5293177366256714, -0.23723222315311432, -0.1873358190059662, -1.3882319927215576, -0.8621565103530884, 1.8576877117156982, 0.49366864562034607, 1.2007107734680176, -0.6161989569664001, -0.9235357642173767, -1.2087270021438599, 1.2909119129180908, -1.0852282047271729, 0.1845371574163437, 0.12919849157333374, 0.7432032823562622, -0.6970576047897339, -0.23899950087070465, -0.987857460975647, 0.2475714534521103, -1.3434921503067017, 0.12974905967712402, -0.7564913630485535, -0.25597700476646423, -0.489481121301651, -0.754895806312561, -0.9378946423530579, 1.0505999326705933, -0.7499253153800964, 0.22814208269119263, -1.8389956951141357, 0.5273361206054688, 0.8372395038604736, 1.0159797668457031, -1.887658715248108, -1.6186187267303467, 0.38409170508384705, 0.5422155261039734, -0.32369333505630493, -1.159572958946228, -0.38541778922080994, -0.7722039818763733, 1.4036791324615479, -1.2961145639419556, 0.3568620979785919, 0.08550100028514862, 1.1588010787963867, 0.3365907669067383, 0.6784924864768982, 0.594049870967865, 0.37689048051834106, 0.17466731369495392, 1.1977571249008179, -0.7636250257492065], [0.5922327041625977, 0.6264227032661438, -0.6850465536117554, -1.7185195684432983, -0.6561091542243958, 1.109183430671692, -0.289286345243454, 0.593920886516571, 2.3555898666381836, 1.1124464273452759, 0.13370674848556519, -1.0980550050735474, 0.18751175701618195, -1.5822875499725342, 0.35470491647720337, -0.1626303344964981, -0.6865533590316772, 1.3916593790054321, -0.03841087967157364, 0.030529312789440155, 1.6186105012893677, -0.895690381526947, 1.2892690896987915, -1.2536234855651855, 0.0035870601423084736, -1.7259153127670288, 1.155884861946106, 0.44642794132232666, 1.07985520362854, 0.07879005372524261, 0.5399985313415527, 0.9484366178512573, -0.09974940121173859, 1.4984838962554932, 0.4647708237171173, -0.427403062582016, -0.37116432189941406, 1.8270212411880493, 0.11448907107114792, -0.9137361645698547, 0.1469387710094452, 0.816145122051239, 0.7836936712265015, -1.8132303953170776, -0.5326910018920898, -0.9126768112182617, 0.3452101945877075, 0.7673307061195374, -0.6220949292182922, 0.7977735996246338, -1.711474061012268, -0.551750898361206, 0.33504053950309753, 0.32288768887519836, -0.8349608778953552, 0.291913241147995, -1.4139928817749023, -0.3755006194114685, 0.8415084481239319, -0.341390460729599, -0.12216711044311523, 0.3120914399623871, 0.7061610221862793, 0.8279312252998352], [-0.07643846422433853, -0.5302411913871765, 0.34165170788764954, 0.24244952201843262, -1.427651286125183, 1.7293363809585571, 1.0654034614562988, 1.1597851514816284, -0.7467707395553589, -0.3784453570842743, -0.6963897943496704, 1.2197757959365845, 0.45540952682495117, -0.2112569808959961, -0.4480772614479065, -0.22974848747253418, -0.2719043791294098, -0.3180564045906067, 0.31079792976379395, 0.1481594443321228, 0.5427871346473694, 0.10610410571098328, -0.7652884721755981, -0.3431043326854706, -0.6479105949401855, -0.2519340217113495, 0.058325137943029404, -1.305591106414795, 0.8112296462059021, -0.24322821199893951, 2.391855239868164, -1.4827723503112793, 1.7506678104400635, 0.306587815284729, -0.829824686050415, 0.53676438331604, -1.6552611589431763, 0.28577056527137756, 0.7196229696273804, 1.2700800895690918, -1.4417619705200195, 0.621139407157898, -0.510811448097229, 1.401720404624939, -0.5261918306350708, -0.28445926308631897, -1.5596516132354736, 0.9982516169548035, -0.7463318109512329, -0.9541842341423035, -0.23749487102031708, 0.4338568150997162, -2.403364658355713, -0.36921805143356323, 0.5000425577163696, 0.35274529457092285, -0.9366788864135742, -1.025968074798584, -0.5491642951965332, 0.5580577850341797, 0.8141991496086121, -0.3688346743583679, 1.175411343574524, -0.8588386178016663], [0.9121710658073425, -1.8697224855422974, 0.8470180630683899, -0.3710499405860901, 1.7173078060150146, -0.11048483848571777, -0.8956781029701233, -1.5762642621994019, -0.21663355827331543, -0.23497329652309418, 0.7342156767845154, -0.5323035717010498, -0.34012967348098755, -1.9663662910461426, -1.4360074996948242, -0.8336606621742249, 1.9170488119125366, 0.2562149167060852, 0.4812726080417633, -1.3415392637252808, -0.6693778038024902, -2.414858341217041, -0.2426121085882187, -0.4147135615348816, 1.3865758180618286, 0.2694684565067291, 0.029130887240171432, -0.19951234757900238, 0.30392077565193176, -0.5677511096000671, -0.06792335212230682, -0.9522857069969177, -0.2754974365234375, -1.8908846378326416, -0.4216265082359314, -0.11078860610723495, -0.6572710871696472, -0.3712913691997528, 0.28623321652412415, 0.4523381292819977, 0.11292712390422821, -0.32577165961265564, -0.5344505310058594, -0.21752718091011047, -1.2278695106506348, 0.41689950227737427, 1.1790785789489746, -0.967377245426178, 0.5109779834747314, 1.743983268737793, 0.6437657475471497, 1.137690782546997, 1.5120882987976074, -1.6238549947738647, -0.43220844864845276, 0.9184818267822266, -0.6458185315132141, -0.8031237125396729, -0.0679580569267273, -0.5213159322738647, 0.49822354316711426, 0.521834135055542, -0.4490163028240204, 1.6444904804229736], [-1.7347919940948486, 0.42064136266708374, -0.10271384567022324, -0.15066732466220856, 0.4600850045681, 0.07631784677505493, -0.7932934761047363, -1.932717204093933, -0.5857849717140198, -0.35583245754241943, -0.1773277223110199, 0.014959965832531452, -0.21210776269435883, -0.4099498689174652, 1.0762126445770264, -0.3409259617328644, -0.13433164358139038, -0.010996835306286812, -0.9340896010398865, -0.31607910990715027, -0.2087147980928421, -1.532175064086914, -0.0873856246471405, 0.3599890470504761, -0.6912358403205872, 0.5389108657836914, 0.5612399578094482, 0.01183610875159502, -0.4941582679748535, -0.30527153611183167, 0.9843884110450745, 0.5460763573646545, 1.3836780786514282, -0.43284377455711365, 0.6230171322822571, 1.1484650373458862, 1.3301299810409546, -1.0374023914337158, 1.3414655923843384, 0.23748834431171417, -1.5761373043060303, -0.4041447937488556, -0.5118862986564636, 1.1584115028381348, -0.8514979481697083, -0.877284049987793, 0.474088579416275, 0.42619389295578003, 0.38843223452568054, 0.3294883668422699, 1.6427172422409058, 0.7744891047477722, 0.32942888140678406, 0.7150665521621704, -1.7719433307647705, -0.8490006923675537, 0.37798410654067993, -0.3280564546585083, 0.3249850571155548, -1.1328182220458984, 0.8982412219047546, -0.02514340914785862, 0.5335478782653809, -0.7428299784660339], [0.06457331776618958, -0.1994272619485855, 2.2613866329193115, 1.5419503450393677, -0.8201228976249695, -0.6740224361419678, -0.4151393175125122, 0.261669397354126, -0.011157839559018612, -1.431599497795105, 1.7048001289367676, -0.5725399255752563, -0.5022059082984924, -1.8619526624679565, 0.3161018192768097, -0.662222683429718, -1.1342785358428955, -0.8940650224685669, 1.0914210081100464, 2.6954829692840576, 0.16561153531074524, 0.16625164449214935, -0.6790481209754944, 1.3472676277160645, -1.8372735977172852, -0.08151786029338837, -1.2933794260025024, 0.7507337331771851, 0.2303379476070404, -0.1782250702381134, -0.7119582295417786, 0.01949148066341877, -0.6603864431381226, 0.12218543887138367, 0.42739465832710266, -0.03094048984348774, -0.08169155567884445, -0.7377248406410217, 0.6877478361129761, -0.4191867411136627, -0.04662839695811272, 0.4064171314239502, 0.9050070643424988, -1.8526865243911743, 0.2667809724807739, -1.5170011520385742, -0.6428040862083435, -1.231095552444458, 0.6917949318885803, 0.44877928495407104, -0.19487732648849487, -1.0383754968643188, -0.32905083894729614, 0.3177935481071472, -0.32265082001686096, -0.017140772193670273, 0.7163166403770447, 0.21370600163936615, -0.9112184047698975, -0.523932158946991, -0.31516003608703613, -0.9319620132446289, -0.6801753044128418, 0.6498585939407349], [1.0238717794418335, 0.7788713574409485, 1.2397111654281616, -0.41189390420913696, -0.23622000217437744, 0.4870757460594177, -0.30921614170074463, 0.748266875743866, 0.9798657894134521, 0.22655512392520905, 0.12750326097011566, -1.5950055122375488, -0.028119603171944618, 0.8422693014144897, -0.14786259829998016, -1.5699142217636108, -0.4743230938911438, -1.9693564176559448, -0.055226799100637436, 0.08132978528738022, 1.993375301361084, -0.21097332239151, -1.6387690305709839, 1.9461554288864136, -0.46738460659980774, 0.4546186327934265, -0.9842836856842041, 0.3930603265762329, -2.4294145107269287, -0.9393174052238464, -0.7334969639778137, -0.7947771549224854, -0.1793864667415619, -0.8981413841247559, 0.46995487809181213, -0.7048335075378418, -0.008364656940102577, 0.13039004802703857, -1.1496800184249878, -0.12620197236537933, 0.3192109763622284, -0.19601556658744812, 0.9068203568458557, 1.4960733652114868, -0.6400348544120789, -0.7947099804878235, -2.020081043243408, 1.0182093381881714, 0.6305057406425476, -0.6810384392738342, -0.3521021902561188, 0.3479326367378235, 1.2611885070800781, 0.7987715005874634, 1.8116259574890137, 0.1907544732093811, 0.0557306744158268, 0.4576549828052521, -0.3794468343257904, -0.9859453439712524, -0.35261788964271545, -1.06191086769104, 1.0780816078186035, -1.0311652421951294], [1.12384033203125, -0.7495864033699036, 0.5892605185508728, -1.2285094261169434, -0.24682995676994324, -0.5759652853012085, 0.9452037215232849, 0.073404461145401, 0.7414551377296448, 0.41173896193504333, 0.392234742641449, -1.096577525138855, 1.1815755367279053, 1.1083440780639648, 0.9017139077186584, 1.3127949237823486, -0.07508144527673721, -1.3680974245071411, -2.149998426437378, 0.20532774925231934, -0.9353547692298889, 0.46892258524894714, -1.9009968042373657, -0.32273441553115845, -0.49729180335998535, -0.6826946139335632, 0.6873370409011841, 1.7846157550811768, -0.016620177775621414, -0.09543890506029129, -1.2692813873291016, 0.39765048027038574, 0.0979791134595871, 0.17210978269577026, 1.2196506261825562, 1.13749361038208, -0.6967460513114929, 1.2916232347488403, -0.3594575822353363, -0.6949461102485657, -0.6892519593238831, 0.09522760659456253, 0.09233061224222183, -0.17563846707344055, -0.13992679119110107, 0.3523750603199005, 0.28090140223503113, 0.7460839748382568, 0.7863891124725342, -0.6380444765090942, 0.08725400269031525, 0.8026804327964783, -0.20653770864009857, 0.5575496554374695, 0.4460804760456085, 1.0002262592315674, 0.07822073996067047, -0.4766212999820709, -0.769562840461731, -0.5609374642372131, -0.017403336241841316, 0.4785490036010742, 1.8346537351608276, 0.6038517951965332], [0.8484467267990112, -1.7313672304153442, -0.6482890248298645, 0.9864692687988281, -1.1448876857757568, -1.3313149213790894, 1.1189727783203125, -2.541611909866333, 0.9564023017883301, 0.5721234679222107, 0.9504008889198303, 0.7759802937507629, -0.5721909999847412, 0.5956831574440002, 0.1623062789440155, 0.18144044280052185, -0.11846987903118134, -1.650055170059204, -0.05490327626466751, 0.8339741826057434, -1.6510257720947266, -0.009604464285075665, 0.5624570846557617, -0.15682001411914825, 0.9277768731117249, -0.18298795819282532, -0.3190658986568451, 0.15163275599479675, 0.7163876891136169, 0.3890140950679779, -0.5752344131469727, -0.0603264719247818, -2.1857872009277344, 0.2488415390253067, 0.7694753408432007, 0.6673725843429565, -0.5241472125053406, 1.3183945417404175, 1.9845869541168213, -1.130161166191101, 1.5544390678405762, 0.3567381501197815, 0.9434272050857544, -0.9597225189208984, 1.9982686042785645, 0.35596251487731934, 0.026139868423342705, -0.4658994972705841, 0.5004696249961853, -1.8470646142959595, -1.2392300367355347, -0.3209058344364166, 0.05654871463775635, 0.50039142370224, 0.9468429088592529, -0.6425763368606567, -0.290729820728302, -0.6962442994117737, -0.12722881138324738, 1.3923975229263306, 0.8624204993247986, 1.7753596305847168, 1.6941215991973877, -1.240155816078186], [-0.37429964542388916, -0.8207556009292603, -1.3837205171585083, 0.21951726078987122, -0.6711219549179077, 1.938816785812378, -0.5869698524475098, 0.8294187784194946, 1.1702995300292969, -0.21408799290657043, -0.011384238488972187, -0.36465588212013245, 0.11896646022796631, 1.20395827293396, 1.2172653675079346, -0.5271774530410767, -1.2290618419647217, 1.1912810802459717, 0.7814610600471497, -1.1062504053115845, -0.6149962544441223, -1.2102673053741455, 0.26962172985076904, -0.7217689156532288, 0.21460558474063873, -0.681052565574646, -1.404036521911621, -0.5779664516448975, -0.6138234734535217, 1.004839539527893, 0.06609050184488297, -2.0774519443511963, 1.6426668167114258, -0.48052236437797546, -0.49808788299560547, -0.2688993513584137, 0.37424346804618835, 0.7345142960548401, 0.05595334246754646, -1.20313560962677, -1.3688915967941284, 1.1174196004867554, 0.665109395980835, 1.1101018190383911, -0.6297733783721924, 0.933162271976471, 1.0473918914794922, -1.6122428178787231, 1.3479493856430054, 1.13944673538208, -0.2548297643661499, -1.2260431051254272, 1.1182432174682617, 0.40892723202705383, -0.803179144859314, -0.7968330383300781, 0.4320594072341919, 1.4692476987838745, -0.9710763692855835, 0.9596989154815674, 0.2493952512741089, 0.28475841879844666, -2.2145628929138184, 1.468043565750122], [0.7383832335472107, 0.755781888961792, -3.246945381164551, 0.4554753005504608, -0.428194135427475, -1.7739497423171997, -0.3164581060409546, -0.755276620388031, -0.5350807905197144, 0.3458308279514313, 0.4910055696964264, -1.4227015972137451, -0.027397459372878075, -2.9673264026641846, -0.40495866537094116, -0.755231499671936, 0.015234118327498436, -0.8022575974464417, -0.16326646506786346, 0.8354804515838623, 0.8010057806968689, 0.4693659245967865, 1.3773791790008545, -0.09651222079992294, -0.37809455394744873, -0.22616924345493317, 0.608921229839325, 0.28759104013442993, -0.7953848242759705, 0.4167473316192627, 1.0069913864135742, 0.6217825412750244, -0.9069074392318726, -2.124429702758789, 0.6398601531982422, -0.9008825421333313, 0.7138126492500305, -0.5433061718940735, -0.9906967878341675, 1.4774240255355835, 0.42359668016433716, -0.2579573690891266, 0.15783452987670898, 1.3201812505722046, 1.6798025369644165, -0.2969376742839813, -0.7576841115951538, 1.1221394538879395, -0.6785874366760254, 1.1914759874343872, -1.1406879425048828, 0.4391271770000458, -1.2214070558547974, 0.15281137824058533, -1.4235836267471313, -0.5304569602012634, -0.35936373472213745, 1.668015480041504, 0.12345018982887268, -0.4915415942668915, 1.5753512382507324, 0.32553282380104065, -0.2364005595445633, 0.5912997126579285], [-0.6824302673339844, 0.19709789752960205, -1.96023428440094, 0.5427684783935547, 0.10646390169858932, -0.8520039916038513, -0.9090617299079895, -0.45059144496917725, -0.4020235240459442, -1.2363948822021484, 0.5188056230545044, -3.8715531826019287, 0.5938013195991516, -1.3061593770980835, 0.5445712208747864, -0.47324857115745544, 0.1893807351589203, -2.0456955432891846, -1.2851697206497192, 0.5676352977752686, 1.109275221824646, 0.3646906912326813, 0.5997341275215149, -0.5896102786064148, 0.7426097989082336, -1.7233161926269531, -0.9742880463600159, 1.4909642934799194, -0.17038913071155548, 0.7283751368522644, 0.5071291923522949, 0.46484553813934326, -1.0284347534179688, 1.0948586463928223, -0.4103696048259735, 0.6661603450775146, 0.09780588001012802, -0.48835280537605286, 0.835673451423645, 0.5954570174217224, 1.7346093654632568, -1.2230994701385498, -2.2208008766174316, 0.2306925505399704, -2.0092415809631348, -0.19997070729732513, 1.3870986700057983, 0.4941834807395935, -1.4424114227294922, 0.3186054825782776, -1.021288275718689, -0.4348908066749573, 0.3734274208545685, 0.44926634430885315, 0.23391450941562653, -0.09307790547609329, 0.5408126711845398, 0.14135728776454926, 0.07164545357227325, -0.24963778257369995, -0.9896102547645569, -0.5839765667915344, 0.3636200726032257, 0.6555938720703125], [-2.3701798915863037, 0.7492587566375732, 1.5382035970687866, 0.8088371157646179, -1.5011204481124878, 1.6690975427627563, 1.5687413215637207, 0.05573587864637375, 1.9126701354980469, 2.203420639038086, -0.8067795038223267, -0.24776241183280945, -0.23503457009792328, 0.1052129790186882, -0.7551894783973694, 0.36631444096565247, -0.2384031116962433, 0.9916038513183594, 0.08804873377084732, -2.179363250732422, -0.8330172896385193, 0.016952652484178543, -0.46562907099723816, -0.7807731628417969, -0.4417180120944977, -1.3240721225738525, -0.025800390169024467, 2.432581901550293, 1.4995758533477783, 0.2199157476425171, 1.056666374206543, -0.2915038466453552, 0.5022757053375244, 0.4683311879634857, -1.4630714654922485, -1.599327564239502, -2.494103193283081, -0.7620337009429932, 0.5809628367424011, 2.2479076385498047, 0.0690922662615776, 1.3580772876739502, -0.020513158291578293, 0.4359023869037628, -2.548637866973877, 1.9507451057434082, -1.0103518962860107, -0.47945648431777954, 1.80027174949646, 2.688474655151367, -1.0795722007751465, -1.3721565008163452, -2.399527072906494, -0.058480918407440186, 0.6452749967575073, 1.918572187423706, 0.10812529921531677, -0.2037602812051773, 0.699316143989563, -1.1880848407745361, 0.40250107645988464, -0.8395612239837646, 2.3920400142669678, -0.8245968222618103], [-1.4338775873184204, 0.9474197030067444, -1.7382644414901733, -0.4767570495605469, -1.105057716369629, -0.4078924059867859, -0.2245820015668869, 2.5887255668640137, 1.2370637655258179, 2.6573879718780518, 0.22693319618701935, 0.8686464428901672, 0.3103811740875244, -0.6061629056930542, -1.4731199741363525, -0.18779464066028595, 1.2343436479568481, -1.6602522134780884, -0.4334962069988251, -0.031722791492938995, 0.48825863003730774, -0.9701625108718872, -0.25084105134010315, -0.5765826106071472, -1.2058008909225464, 1.0766979455947876, 0.5098760724067688, 0.6288645267486572, 1.3475418090820312, 1.9998449087142944, -0.1323195844888687, -2.2918968200683594, 0.3751107454299927, -1.0906058549880981, -0.7520533800125122, -0.3288569450378418, -0.8441423177719116, -0.4173915982246399, -0.3935029208660126, 0.44216954708099365, -0.8399522304534912, 2.0270206928253174, -0.4309593141078949, -1.154415249824524, 0.18515121936798096, -0.33860674500465393, 1.4984054565429688, -0.10079646855592728, 1.1316462755203247, 1.6709309816360474, -0.6956048607826233, 0.43869855999946594, 1.4775090217590332, -1.528879165649414, -0.08039738237857819, -0.2177417278289795, -1.15371572971344, -2.4159960746765137, -0.24040277302265167, -2.479020833969116, -0.5030151009559631, -1.0215179920196533, 1.0957083702087402, 0.30008506774902344], [-0.5246527791023254, -1.7036211490631104, -0.1917111873626709, -1.1885361671447754, 0.22017154097557068, -0.43398427963256836, -1.1429073810577393, -0.16464997828006744, 1.9113985300064087, -0.38298842310905457, 2.328712224960327, 1.0358401536941528, 1.59602689743042, -0.19512644410133362, 0.33628231287002563, 0.10391805320978165, 1.4418319463729858, 1.1391997337341309, 0.6670161485671997, 0.1993347853422165, -0.20611166954040527, 0.0018778136000037193, 0.36358287930488586, -1.6075868606567383, -0.7252486348152161, 0.9284574389457703, -1.3031960725784302, -0.09303924441337585, 1.082943081855774, 0.5229235887527466, -0.33245009183883667, -0.28750747442245483, -0.736371636390686, 0.053925927728414536, 0.5875146389007568, -0.7655054330825806, 0.008504960685968399, -0.15991318225860596, 0.677639365196228, 0.4039185643196106, -0.6858532428741455, 1.108250379562378, 2.884855270385742, 0.06696375459432602, 1.2093210220336914, -2.5997955799102783, -1.3141416311264038, 0.7677600383758545, -0.09137896448373795, -0.38661524653434753, 0.35902997851371765, 1.5049095153808594, -2.063347101211548, -0.033408213406801224, -0.9228391647338867, 0.603283703327179, 0.5811112523078918, -0.1960407942533493, -1.1870852708816528, 0.018306288868188858, -0.7515113949775696, -0.8787884712219238, -2.008932113647461, -0.09077535569667816], [0.9801170229911804, 0.5356676578521729, 0.1615203469991684, -0.8167378902435303, -1.0284638404846191, -0.6206254363059998, -0.2605752944946289, 2.056168794631958, -0.41890421509742737, -0.480172336101532, -2.136533260345459, 0.5407276749610901, -0.43524983525276184, -0.44840285181999207, -2.7324013710021973, -1.0494019985198975, 0.8542423248291016, -0.5807686448097229, 0.16786004602909088, 0.43729937076568604, -1.9270687103271484, 1.7433730363845825, -0.7033680081367493, 1.0041731595993042, 1.8331347703933716, -1.4337408542633057, 0.1309802532196045, 1.1358481645584106, -1.638979434967041, -0.7192376255989075, -1.1024638414382935, 0.07027407735586166, -0.011882559396326542, 0.05439334362745285, 0.6959264874458313, 0.18575407564640045, 0.772470235824585, -1.4333642721176147, 2.027690887451172, -1.6284819841384888, -1.2762620449066162, 0.24146167933940887, 0.2963124215602875, 0.6362927556037903, -0.6296754479408264, 0.3330918848514557, -1.2540698051452637, 0.11448018997907639, -2.289226531982422, -1.1936330795288086, 0.11897266656160355, 0.06287757307291031, -0.8865491151809692, -0.36562275886535645, 0.9754201769828796, 1.4601281881332397, -1.088499903678894, 2.0984625816345215, 1.7658416032791138, -0.7967742681503296, -0.10325634479522705, -0.1692689210176468, 0.10667320340871811, -0.31598955392837524], [2.3308892250061035, -0.052045028656721115, -1.6805213689804077, -0.8181732296943665, -1.776132583618164, 0.012658283114433289, 0.8360989689826965, 0.3511345088481903, 0.38444849848747253, -0.7861881256103516, -0.41593924164772034, -1.5255705118179321, -1.3093857765197754, -2.8486297130584717, 0.5565888285636902, 0.32821065187454224, -1.140625, 0.06885554641485214, 0.9610286355018616, -0.8167920708656311, 1.457257866859436, 0.05335683375597, 0.5453574657440186, 2.566751718521118, -0.158767968416214, -0.946837842464447, 1.0340731143951416, 0.03438466787338257, 0.5073980093002319, -1.5440682172775269, 0.39865243434906006, 1.178060531616211, 0.35714101791381836, -0.41675320267677307, -1.0488311052322388, 0.34848377108573914, 0.29326173663139343, -0.1508565992116928, -0.5633591413497925, -1.3732774257659912, -0.03550459071993828, -1.0036920309066772, -0.7119040489196777, -1.1618621349334717, -3.557814121246338, -0.5074558854103088, -0.6022866368293762, -0.5572288632392883, -0.17671699821949005, 0.04559342563152313, 1.8750380277633667, -0.6224676966667175, -0.6345035433769226, -0.4516657888889313, -1.728696584701538, -0.3692842125892639, 2.2147562503814697, 0.4005652666091919, 1.0377296209335327, 1.881099820137024, 3.6032936573028564, -0.6865176558494568, 0.30524352192878723, -1.1341354846954346], [-1.346672773361206, -0.24879422783851624, -1.9251364469528198, -0.022385049611330032, -1.0359691381454468, 0.36364734172821045, 0.7490326166152954, -0.2099427431821823, 0.834503173828125, -1.2977180480957031, 1.4851678609848022, 0.5209059715270996, 0.5487911105155945, 0.24982291460037231, -0.5993256568908691, 0.9311736822128296, 1.1505975723266602, 2.2625701427459717, -0.5119373202323914, 0.27495425939559937, 0.40402400493621826, -0.7458689212799072, 0.9029830098152161, -1.3591912984848022, 0.0719916895031929, -0.7561771869659424, 0.9177049398422241, 1.075736165046692, -1.31453275680542, -0.3625951409339905, -0.0684911459684372, -0.8127951622009277, -0.5123007297515869, -0.056803859770298004, 0.6871559619903564, -1.5284068584442139, 0.10895941406488419, 0.6389067769050598, 1.8723201751708984, -0.5613066554069519, -0.6212451457977295, -0.5482680797576904, 1.0008918046951294, 1.0749739408493042, 1.700238823890686, -0.3066267669200897, -0.40483781695365906, -0.6927916407585144, -0.7350267767906189, 1.6158077716827393, 1.2819403409957886, -0.0023875038605183363, -0.5896508693695068, 0.11631140112876892, -0.8557595014572144, 1.8773926496505737, -0.3244352340698242, 0.5968384146690369, 1.1007072925567627, -1.2144269943237305, -0.44898509979248047, -0.06524330377578735, -1.1834282875061035, 1.9399561882019043], [2.138568639755249, 2.529618740081787, 0.46438467502593994, 2.1143622398376465, -1.4023220539093018, -0.2109011709690094, 0.7611561417579651, 0.2914925217628479, -0.4802217483520508, -0.10315470397472382, -1.633325457572937, -0.4462162256240845, 0.3316037952899933, 0.1491735279560089, 0.026758162304759026, 0.9813718795776367, 0.41927725076675415, 1.4873076677322388, 1.1635735034942627, -0.7616055011749268, -0.5069673657417297, -0.45603543519973755, 1.4679384231567383, 0.4759595990180969, -1.5769574642181396, 1.1155571937561035, -0.39389944076538086, -1.0444318056106567, 0.9114007949829102, -0.24561458826065063, 1.089438557624817, 0.13414110243320465, -0.33418190479278564, 0.38559284806251526, 0.05824096500873566, -0.8441526889801025, -2.4967894554138184, -0.10095013678073883, 1.2064082622528076, -0.7988232970237732, -2.591852903366089, 0.42505335807800293, -0.19800370931625366, 1.0140678882598877, -1.7655829191207886, -2.949964761734009, -2.4258370399475098, -0.19375699758529663, -0.39297014474868774, -1.0438458919525146, -0.22682572901248932, 2.028343915939331, 1.466477870941162, 0.4070083498954773, 1.1014608144760132, -1.6297560930252075, -1.508943796157837, -1.433449387550354, 0.23526236414909363, 0.10164317488670349, -0.28252631425857544, 0.056622158735990524, -0.39956772327423096, -2.144392728805542], [0.715505838394165, -1.752246379852295, 0.3850642740726471, 0.038636207580566406, -0.9658116698265076, -0.0369635634124279, 0.5538123846054077, 0.21614596247673035, -0.8441363573074341, -0.33783993124961853, -1.4724586009979248, -1.037165880203247, 0.25045132637023926, -0.1344471275806427, 1.1290500164031982, -0.3276464641094208, -0.5286402106285095, 1.6272841691970825, -0.19967690110206604, 0.20490814745426178, 1.2590539455413818, 0.5574514865875244, -0.7036347985267639, 0.7062206268310547, -2.2407021522521973, 0.22606684267520905, 2.117870330810547, -1.22916841506958, 0.22784832119941711, -0.19854964315891266, -0.10605408996343613, -0.13906551897525787, -0.4804043769836426, -0.660188615322113, -1.278454065322876, 0.31207019090652466, -0.6412030458450317, 1.937739610671997, -1.5346647500991821, 0.788567841053009, -0.3715403079986572, 0.41756680607795715, 0.5313356518745422, -0.3272722661495209, -1.4015088081359863, -1.1984256505966187, 1.0895335674285889, 0.9582486748695374, -0.9127853512763977, 0.013260338455438614, 1.136417269706726, 0.9613130688667297, -0.36431390047073364, -0.18129941821098328, 0.8082231283187866, -0.11593388766050339, 0.5927306413650513, 0.11098510771989822, 0.9329010248184204, -1.310915470123291, -2.2363178730010986, -0.18371635675430298, -0.5611841082572937, 0.6846752166748047], [0.6951992511749268, 0.46281319856643677, -1.1977579593658447, 1.8720972537994385, -1.3133032321929932, 0.571677565574646, -1.4411686658859253, 0.6624361872673035, 1.8546397686004639, -1.6371830701828003, -0.9578689336776733, -0.06030834838747978, -0.20890673995018005, -1.3409737348556519, -0.0267073642462492, 0.5294127464294434, -2.238248109817505, -0.804313600063324, -0.7666308283805847, 0.28975677490234375, 1.095172643661499, 0.7806397676467896, 0.6334023475646973, -0.6461226344108582, 0.2624526917934418, -0.5429812073707581, 0.22823722660541534, -1.8267433643341064, -0.8559398651123047, 0.5498171448707581, -0.9295098781585693, -0.6887280344963074, 1.765748381614685, 0.06785736978054047, -0.5667873620986938, 0.16222701966762543, -0.056877557188272476, -1.3590742349624634, 1.5990548133850098, 1.8227301836013794, -0.14435431361198425, 0.48397794365882874, -0.22613859176635742, 1.6859880685806274, -0.4815153479576111, -0.22374227643013, 2.3391332626342773, 0.6978164315223694, 0.4113698899745941, 0.9550413489341736, 0.34874746203422546, -1.0986889600753784, -0.4986099898815155, -0.9363058805465698, 0.39800336956977844, -0.05878845229744911, 0.6555118560791016, -0.4568135142326355, -0.5165349841117859, -0.3717317283153534, 0.5432456135749817, -0.11111025512218475, 1.4568097591400146, -0.43309998512268066], [-0.5493677854537964, -0.8268852233886719, -0.2415868192911148, 0.13725335896015167, -0.4240654706954956, -1.0230333805084229, 0.024343006312847137, 0.026109471917152405, -1.2412934303283691, -1.3465876579284668, -1.081019401550293, -0.3590945899486542, -0.4442784786224365, -0.45214560627937317, -0.11135534942150116, -0.6545596122741699, 0.1357969343662262, -1.100113868713379, -0.7840703725814819, 1.9798004627227783, 0.018558211624622345, 0.6422070860862732, 0.7481624484062195, -0.9974209666252136, -0.05568113550543785, 0.359973669052124, 1.0356147289276123, 1.1906213760375977, 0.47870728373527527, 1.5358171463012695, 0.9094926714897156, -0.33766311407089233, -0.09916332364082336, 0.9741923213005066, 1.4277231693267822, 0.3229339122772217, 0.3321690559387207, 1.34078049659729, -0.1467963308095932, 1.6790494918823242, -1.6182233095169067, -0.6715390086174011, -0.36961254477500916, -1.8480404615402222, -1.4388067722320557, -0.6222351789474487, 2.7021968364715576, -0.2774590253829956, -0.10738275945186615, 1.6045717000961304, -0.26909583806991577, 0.002811175538226962, -0.18647800385951996, 1.3348687887191772, 1.1300761699676514, 1.2092341184616089, -0.6202205419540405, -0.6996016502380371, -0.23429377377033234, 0.7756430506706238, 1.381636381149292, -0.4466843605041504, -1.3851115703582764, -0.38524284958839417], [0.01538076438009739, -0.858694314956665, 0.7497571110725403, 0.7014754414558411, -1.6975657939910889, 1.1629047393798828, -0.9320370554924011, -1.905611515045166, -0.6598771214485168, 1.7612518072128296, 1.5839260816574097, 0.29434192180633545, 0.7229270935058594, 0.5656264424324036, 1.189781665802002, 0.20422109961509705, 0.139853373169899, 0.7318011522293091, 0.3589046895503998, -1.1986541748046875, 1.187610387802124, -1.0458844900131226, -0.15819287300109863, 0.43936654925346375, 1.3146425485610962, -1.6263278722763062, 0.9061861634254456, -0.15532155334949493, 0.10549817234277725, -0.8791831731796265, 0.02040274254977703, 0.39078834652900696, -1.1747205257415771, 0.8053402304649353, 1.1044703722000122, -0.715429961681366, 1.6923726797103882, -0.7957762479782104, 0.5050451755523682, 1.4700425863265991, -0.6146606802940369, 0.9287894368171692, -2.267975330352783, -0.4627918303012848, 0.39030691981315613, -1.1227794885635376, 0.3181719183921814, -1.6902257204055786, 0.44258207082748413, 0.732994794845581, 0.12317706644535065, 0.31635016202926636, -0.38349178433418274, -0.9625464677810669, 0.6000862121582031, 1.0746450424194336, -0.33233842253685, 1.7736483812332153, 1.0141645669937134, -2.4536898136138916, 1.3934110403060913, 1.0829145908355713, -0.2819859981536865, -0.8092754483222961], [0.5623318552970886, -1.095821738243103, -1.2488865852355957, 0.8254672884941101, 2.037961721420288, -1.1248730421066284, 1.0127607583999634, -1.0348737239837646, 0.8524417281150818, -1.794719934463501, 0.7791869044303894, 0.06263729929924011, 0.9476461410522461, 0.4333563446998596, 0.5144490003585815, 0.09536901116371155, 0.41945186257362366, -0.2784794569015503, 0.15808403491973877, 0.20030370354652405, 1.2433278560638428, -0.4778389632701874, -0.03871162235736847, 0.06721401959657669, -0.2785710394382477, -1.022130012512207, -0.9595975279808044, 1.3132468461990356, 0.562444806098938, 0.4943363070487976, 0.6274605989456177, 0.2838984429836273, 0.9838637113571167, -1.3678041696548462, -1.3742958307266235, 0.18212352693080902, -0.2936115860939026, -0.16402730345726013, -0.32082629203796387, 0.4924420416355133, -2.2898406982421875, -0.24860839545726776, -0.18103399872779846, -0.04260662570595741, 0.8623659014701843, -1.6057184934616089, -0.5358677506446838, 0.47275128960609436, -0.22842086851596832, -0.4213697016239166, -0.11391621083021164, -0.5938611626625061, -0.6118149161338806, -0.6589315533638, -1.3361376523971558, -0.0934496521949768, 0.7290632128715515, 0.32809048891067505, -0.49821701645851135, -0.5756702423095703, -1.1302964687347412, 0.8589562177658081, -0.9274961948394775, 0.3345281779766083], [0.062499307096004486, 0.07351990789175034, -1.0954235792160034, -0.5072064995765686, 0.9597567319869995, -0.5435839891433716, 1.629758358001709, 0.30484962463378906, -1.204943060874939, 1.9302287101745605, 2.315650701522827, -0.3374882936477661, 0.5577988028526306, -0.6136465072631836, 0.8085301518440247, 0.34259602427482605, 1.9584256410598755, 1.5179359912872314, -1.957723617553711, 0.09640239179134369, 0.4051721394062042, 0.5801198482513428, -0.04267359897494316, -0.3200564682483673, 0.5055312514305115, 0.8234522342681885, -0.8052430748939514, -0.2344038486480713, 0.9859871864318848, 0.13811039924621582, 0.3087748885154724, -0.44629666209220886, 1.9915395975112915, 1.6001601219177246, -0.11985211819410324, -0.5131769180297852, -1.8523118495941162, -0.26685506105422974, -0.35394027829170227, 0.44988131523132324, -1.0037696361541748, -2.1194345951080322, -0.6403851509094238, 1.1928402185440063, -1.9404847621917725, 0.14036892354488373, -0.46928033232688904, -0.8434157967567444, -1.6673829555511475, -0.27013829350471497, -1.0122894048690796, -0.7463486194610596, 0.5550820231437683, 0.9750367403030396, -0.3611619472503662, -1.0347867012023926, -1.8144925832748413, -0.11148107051849365, 0.2985122501850128, -0.40464121103286743, -0.037005241960287094, 0.6801983118057251, 2.387453556060791, 1.328622579574585], [1.0411113500595093, 1.2368605136871338, 0.16189023852348328, -0.3418252766132355, -0.21065860986709595, -1.008394479751587, 0.8451457619667053, 0.5583608150482178, -1.3290125131607056, -0.580288827419281, 0.875852108001709, -0.13530565798282623, -0.8293176293373108, 0.47724977135658264, 0.44235146045684814, -0.32854655385017395, -1.5314122438430786, 0.6424692273139954, 0.4014366865158081, -1.0364185571670532, -0.8723504543304443, 1.101836085319519, -0.10789979249238968, -1.1541649103164673, -0.46622514724731445, 0.13118213415145874, 0.05705804377794266, 0.29699066281318665, 1.0665227174758911, -1.7670156955718994, -0.8641179203987122, -1.6037993431091309, -0.9954381585121155, 0.6365240812301636, -0.5403599143028259, -0.9411320090293884, -0.5157409310340881, -0.21222884953022003, -0.7002253532409668, -0.9558069705963135, -0.37591320276260376, 0.6885045766830444, -0.899572491645813, 0.3172757029533386, -1.568534016609192, 1.276053786277771, 1.0480637550354004, -1.5400718450546265, -2.5020291805267334, -0.8550244569778442, 0.23812292516231537, 0.686922013759613, -0.7963463664054871, -0.6996133327484131, 1.3386728763580322, 1.2733826637268066, 0.7559729814529419, 0.45043304562568665, 2.769543409347534, 1.1924978494644165, -1.47040855884552, -0.8376295566558838, -1.1063778400421143, 0.9022486805915833], [-0.17710773646831512, 0.7238727807998657, -0.37131208181381226, 0.14889740943908691, -0.020787931978702545, -1.154971957206726, -0.0428885854780674, 1.097348928451538, -0.3238489627838135, -0.04083976149559021, 0.9494667649269104, 0.3474792242050171, 0.9583002924919128, 0.16678862273693085, -1.3159552812576294, 0.296149343252182, 0.47927623987197876, -0.8156188130378723, 0.2222771942615509, 1.4109607934951782, 0.4021550118923187, 2.080390214920044, -1.05836021900177, 0.17933066189289093, -0.7098841071128845, 0.3888590335845947, -1.1835582256317139, -1.1691834926605225, 0.0281304270029068, -0.5814647674560547, 0.6823885440826416, 0.6461623311042786, 1.0424190759658813, -0.2460150569677353, -0.2507181763648987, -1.6600372791290283, 1.1090774536132812, 0.13900920748710632, -1.302345871925354, -0.5372616052627563, 1.2830417156219482, 0.960034191608429, -0.1527099460363388, -0.21885289251804352, -1.0697015523910522, -0.41531893610954285, -1.0590741634368896, 0.11710762232542038, -0.9690321087837219, -1.7780935764312744, -0.5979713797569275, -1.5754482746124268, -1.3258181810379028, -1.1701843738555908, 1.6279984712600708, -1.339686632156372, 1.9944260120391846, 1.106284499168396, 0.12880216538906097, 0.5818928480148315, -1.805465817451477, -0.0071108378469944, -0.03687494248151779, -1.382655143737793], [-0.8312150835990906, -0.9064356684684753, -1.7596378326416016, 0.9562456011772156, 0.5608194470405579, -0.3241991698741913, 0.02450510673224926, -0.789819598197937, 1.806530475616455, 1.19576096534729, 3.239607095718384, 0.32762807607650757, 0.5552176237106323, 0.48510298132896423, -0.11232851445674896, 0.6844933032989502, 0.12600578367710114, 2.4630887508392334, 0.13331028819084167, -1.157943606376648, -1.5700526237487793, 0.5433464050292969, -0.8822067379951477, -0.4681686460971832, -0.9303120374679565, 0.4579615592956543, 0.8457500338554382, 0.035949282348155975, 0.010250051505863667, 0.8754897713661194, -0.5542566180229187, 0.9515160322189331, 1.0751899480819702, 1.090352177619934, 1.9813851118087769, 0.9387547373771667, 0.607979416847229, 0.8238083124160767, -0.2814226448535919, 0.9253915548324585, 2.074735403060913, 0.0004151690809521824, -1.1894010305404663, 0.22980347275733948, -0.9219200611114502, -0.32118263840675354, 1.4919884204864502, 0.11999941617250443, 0.5083590745925903, 0.21275971829891205, 0.40491336584091187, 0.5625290870666504, 0.21464350819587708, 0.4844796061515808, 0.8140097856521606, -0.9056694507598877, -0.8799754977226257, 0.8397310376167297, 0.08335371315479279, -1.7059038877487183, -0.32224801182746887, -0.1796495020389557, 0.6816682815551758, 2.117664098739624], [-1.5193510055541992, 0.6920382380485535, 0.05958465486764908, -1.2192976474761963, -0.011129370890557766, 0.18706469237804413, -1.313804268836975, -1.420499563217163, 0.5458939671516418, -0.7822664976119995, -1.378771185874939, 0.47107380628585815, -1.121970295906067, 0.8529553413391113, 0.9744964241981506, -0.06977570056915283, 2.0278196334838867, 0.20225295424461365, 0.9245244264602661, -0.561004102230072, -0.03481857106089592, -1.7901806831359863, 2.432387351989746, -0.554053008556366, 0.27642691135406494, 1.2323499917984009, 1.5296120643615723, -0.025054482743144035, -0.22111932933330536, -1.0390355587005615, -1.663812518119812, 0.3945375680923462, 0.49549758434295654, 0.658330500125885, 0.1719762682914734, 0.7990655899047852, -1.8705588579177856, 1.9223768711090088, -0.5041889548301697, 0.04748218134045601, -0.8479929566383362, 1.396712303161621, 0.3097313642501831, 1.5837767124176025, -0.1024184599518776, -0.4160900413990021, -1.3194202184677124, -1.55390202999115, -0.9108548164367676, -0.8123511672019958, 0.40319085121154785, 0.12078559398651123, -1.3295807838439941, -1.737733244895935, 0.45048993825912476, -0.1410032957792282, 0.5556663870811462, 0.7211721539497375, -0.7060665488243103, -0.9336032867431641, -0.6611071825027466, 0.08470964431762695, -1.6095143556594849, 0.9496892094612122], [-0.16697657108306885, -0.19837890565395355, -0.7965410351753235, 0.933046281337738, -0.8859663009643555, -2.0930027961730957, -0.9265896081924438, -1.5109506845474243, -1.1532864570617676, -0.21325188875198364, 1.7109413146972656, -0.085518017411232, -0.6671139001846313, -0.9114212393760681, -0.38447943329811096, 0.32955101132392883, 0.6441146731376648, -2.1426889896392822, -0.6092126369476318, -2.239572763442993, -0.44606679677963257, 0.680855929851532, 1.123673915863037, -1.4101598262786865, 1.6317633390426636, 1.8422433137893677, 1.1559863090515137, 0.0829487219452858, 0.011628256179392338, -1.3151968717575073, -0.010136707685887814, 0.532787561416626, -0.4365091025829315, -1.3158544301986694, 0.2218637317419052, -0.16838985681533813, -0.38028228282928467, -0.7653048038482666, -0.20230641961097717, 0.7690768241882324, 0.780226469039917, -0.02312539890408516, 0.6273796558380127, -1.0566166639328003, 0.15632708370685577, -0.62981116771698, 1.059027910232544, 1.626341462135315, 0.28195720911026, 0.0606234148144722, -0.21589672565460205, -0.11477617174386978, -0.13209885358810425, 0.7889679670333862, -1.97537100315094, 1.3993428945541382, -0.07840269804000854, -2.025123357772827, -0.7166590690612793, 0.892896294593811, 0.9000822305679321, 0.33432966470718384, -1.0029799938201904, 1.1624749898910522], [-0.6860833764076233, -0.019109684973955154, 0.33197838068008423, -0.2793487012386322, -0.9301544427871704, -0.5293640494346619, 0.4464320242404938, -0.38706234097480774, -0.8541792035102844, -0.6232975125312805, -1.292109727859497, 0.2036582976579666, -0.44710853695869446, -0.5370100736618042, 0.7645963430404663, -0.09703801572322845, -0.2900027334690094, -2.0617504119873047, 0.334994375705719, 0.8108626008033752, 0.5819240808486938, -0.504197895526886, 0.31146112084388733, 0.5740420818328857, 0.25789710879325867, 1.6826839447021484, 0.057775240391492844, -0.06399296224117279, -0.4435535669326782, 0.42569541931152344, -0.9105532169342041, -0.34666281938552856, 0.796847403049469, -1.5881304740905762, -1.4987373352050781, -0.36315837502479553, -0.5034279823303223, -1.9390177726745605, -1.1496633291244507, 0.20357714593410492, -0.8992785215377808, 0.2551566958427429, -1.5358330011367798, 0.380779504776001, -1.2822370529174805, 0.552787184715271, -0.10371917486190796, -0.6001784205436707, 1.398486614227295, -1.5049927234649658, -1.8607933521270752, -1.078561782836914, -0.9380412101745605, -0.4240248203277588, -1.964651107788086, 0.7568750381469727, 0.6934573650360107, -0.5599436163902283, 0.09646357595920563, 0.42632922530174255, 0.4532719552516937, 2.4043033123016357, -0.04628487303853035, -0.22585931420326233], [-0.13446567952632904, -0.9351701736450195, 0.1818254590034485, 0.07956453412771225, 0.7278019785881042, -0.6940001249313354, 0.1869613528251648, 0.6872915029525757, -0.8110734820365906, -1.3131239414215088, -1.9744850397109985, 0.7272435426712036, 0.14582596719264984, 1.0653467178344727, 0.6745524406433105, 1.4390816688537598, 1.708577036857605, -0.25822389125823975, 0.08377294987440109, -0.9357348680496216, 0.33626431226730347, -0.471209317445755, 0.4905346930027008, 1.0037150382995605, 1.3839458227157593, -1.031005859375, 0.3000285029411316, 1.9503459930419922, 0.24224571883678436, -0.8550535440444946, 0.47234365344047546, 0.5677539706230164, -0.6864101886749268, -0.22157412767410278, -1.3772473335266113, -0.9098325967788696, 0.7439501881599426, -0.6977834105491638, -0.36196815967559814, 0.7373763918876648, 0.6530592441558838, 1.5903650522232056, 0.44342154264450073, 2.180485248565674, -0.1829727739095688, -0.03813786059617996, -1.2603672742843628, -0.43185174465179443, -0.9327975511550903, -0.6149932742118835, 0.05616647005081177, -1.1375335454940796, -0.5697922110557556, -1.3329888582229614, 0.7654143571853638, -0.5079997777938843, 1.7684367895126343, 1.2938053607940674, 0.41627711057662964, -2.2997007369995117, 0.35367512702941895, 0.9667690992355347, -1.3990169763565063, -1.3819316625595093], [-0.697984516620636, -0.4100756049156189, 0.6095473170280457, -0.34098178148269653, -1.5755443572998047, -0.03428405150771141, -0.14776954054832458, -0.9143023490905762, 0.5544737577438354, -1.0020323991775513, 1.0178364515304565, -0.6983484029769897, -1.201624870300293, 0.6789441108703613, -0.11469383537769318, -0.8815857768058777, 0.4246973693370819, 1.1587493419647217, -0.4003467857837677, 0.012434824369847775, 0.3399170935153961, -0.20051608979701996, -0.7213008403778076, 0.6341789364814758, -1.3787894248962402, -1.3580681085586548, -1.008583903312683, -0.932384729385376, 1.1976944208145142, -0.6394371390342712, -0.6135847568511963, 0.7377235293388367, 1.5581692457199097, 0.6627007126808167, -0.4651789963245392, 0.704251766204834, -0.271424263715744, 1.0176767110824585, 1.2424553632736206, 1.756842017173767, -0.2243625968694687, 0.129026859998703, -0.5501975417137146, -1.439197063446045, -1.1089283227920532, -0.9787951707839966, 0.49204200506210327, -1.2655280828475952, 1.0567913055419922, -1.9240399599075317, 1.8569895029067993, 1.4985545873641968, -1.0313136577606201, 0.5791858434677124, -0.49368301033973694, -0.9022199511528015, -0.48646068572998047, 0.14081312716007233, 0.7124930620193481, -0.6163381934165955, -0.36568111181259155, -0.3345988392829895, 0.3696877360343933, -1.6102886199951172], [0.8760983943939209, 1.1372148990631104, -1.0095744132995605, -0.813342273235321, 0.8704830408096313, -0.44483649730682373, -0.09709908813238144, 1.5206905603408813, -1.2940949201583862, 2.0631425380706787, 1.0282479524612427, 0.697177529335022, -0.24238653481006622, 0.13198792934417725, 0.8243638277053833, -1.0628600120544434, 0.2752622663974762, -0.2281443476676941, 0.23451371490955353, 1.9600098133087158, -0.9246336221694946, -1.084604024887085, -1.7361375093460083, 0.8073388338088989, 0.040916722267866135, -0.16498546302318573, -0.6063172817230225, -1.7061854600906372, 0.8838502168655396, 0.5008049011230469, -0.8276394605636597, -0.3639553487300873, 0.5199714303016663, -0.693861186504364, -0.3751540780067444, 0.5949172973632812, -0.12796755135059357, -1.172285795211792, -1.4033995866775513, -0.2359907180070877, -1.44638991355896, 0.5906422734260559, 0.1237795501947403, -1.8984795808792114, 1.7721946239471436, 0.6943076252937317, -1.7483162879943848, -0.10346505045890808, 1.0098352432250977, 0.9466057419776917, 1.7644885778427124, -0.5082001090049744, -1.6296335458755493, 0.8719200491905212, -0.1509798914194107, -0.5434533357620239, 0.9886534810066223, 0.9475097060203552, 0.38734695315361023, 0.8528594374656677, 0.5654353499412537, 1.6200690269470215, -0.8987292647361755, -0.053103968501091], [-0.1237480565905571, 0.24592708051204681, -2.0716001987457275, 1.2880228757858276, 0.45730531215667725, 0.7130259871482849, 0.08535666018724442, 0.14695973694324493, 1.2170937061309814, -0.5289551615715027, -0.5222812294960022, 0.13691821694374084, -1.994655728340149, -0.1291119009256363, 1.1239469051361084, 0.37234461307525635, -0.3156015872955322, 3.2670717239379883, -1.0618547201156616, -1.6726109981536865, 0.020126184448599815, 0.9473208785057068, 1.324583888053894, 0.5213417410850525, 0.23846961557865143, 0.764600932598114, -1.0589061975479126, 0.5340001583099365, -1.2807613611221313, -1.3108311891555786, 1.3414214849472046, 0.8862174153327942, 2.159440517425537, -0.2364943027496338, -1.7881793975830078, 1.6836433410644531, 0.8290185332298279, 0.4183935523033142, 0.07072757929563522, 0.9457359313964844, -0.5570319890975952, -1.8539890050888062, -2.211040496826172, 1.2359405755996704, 1.0770387649536133, 0.11741175502538681, 0.08841564506292343, -0.05494745075702667, -1.0423107147216797, -0.18858852982521057, -0.4505266547203064, 1.2973442077636719, 0.8024470210075378, 0.12925085425376892, 0.7759745717048645, 0.19696956872940063, 1.794335126876831, -1.8627005815505981, -0.503605306148529, 1.1472694873809814, -0.024593660607933998, -0.7062205076217651, -0.5390250086784363, 0.4586193859577179], [-1.1077020168304443, 0.09255696088075638, -0.47948014736175537, -0.6603235602378845, -0.3873319923877716, 1.6682605743408203, -0.8729918003082275, 1.3345164060592651, -0.6632179617881775, 0.6576648354530334, -0.6338935494422913, -1.2804912328720093, 1.260067343711853, -0.31098759174346924, 1.1421396732330322, -1.6482715606689453, -2.104497194290161, 0.809893786907196, 0.7611446976661682, -0.24076880514621735, 0.4657544791698456, -1.1307663917541504, -0.18827494978904724, -0.5806608200073242, 0.9131389856338501, 0.5481322407722473, -0.924358069896698, -0.9107017517089844, 1.4307721853256226, 0.09647908061742783, -0.22859661281108856, -0.7974507212638855, -1.5043416023254395, -0.026002222672104836, 0.997829020023346, -0.8024193048477173, -0.5286064743995667, 0.256849467754364, 1.3551561832427979, 0.4607122242450714, -0.30863314867019653, -1.0488855838775635, 0.9008973240852356, -0.07335693389177322, -0.5527931451797485, 1.1999064683914185, 0.893184244632721, -0.23087424039840698, 0.06676580756902695, -1.2202785015106201, -0.6746619343757629, 0.8792492151260376, 0.8545749187469482, -1.3568602800369263, 0.6186541318893433, 0.16123737394809723, 0.8807713985443115, -0.4341985881328583, -2.0466558933258057, -0.46745437383651733, 0.7253615856170654, -0.3623943328857422, -0.09672413766384125, -1.074934959411621], [-0.7360389828681946, 0.30056318640708923, -0.39893582463264465, -0.5662856698036194, -0.3968048393726349, -1.2232459783554077, -0.7243425846099854, 0.3767520487308502, 0.8660255074501038, 0.10047494620084763, -1.1821844577789307, -1.5055986642837524, 0.6566013097763062, 0.13003428280353546, 0.7714963555335999, 0.038171492516994476, -0.48831427097320557, -1.4620702266693115, -1.1251989603042603, -0.035046011209487915, -0.062093570828437805, 0.005743798799812794, 0.5048043131828308, 1.7491106986999512, -0.47737136483192444, 1.6442768573760986, -2.0548717975616455, 1.8140498399734497, -0.862589955329895, -1.5666577816009521, -0.5043671727180481, -1.2872209548950195, -0.6048098802566528, 0.36277076601982117, 0.3062434196472168, -0.6903512477874756, 0.2988002896308899, 0.2584376335144043, -1.6269917488098145, -2.155491352081299, -0.0027009935583919287, 0.34778833389282227, 1.3143954277038574, -1.1132885217666626, -0.8948448896408081, 0.8354928493499756, 1.2606470584869385, -0.3680591881275177, -0.6395886540412903, 0.37936151027679443, 0.5419079065322876, -0.13296885788440704, 1.4720957279205322, -0.09214302897453308, -0.2645339369773865, 1.0743021965026855, 0.22978061437606812, -0.1844770908355713, 0.12163659930229187, 0.6823031306266785, 0.19207821786403656, 1.7787284851074219, 0.6263995170593262, 0.47533729672431946], [-0.31922465562820435, -1.1831015348434448, 0.5834028124809265, 1.3930937051773071, -0.2952360510826111, 0.7016892433166504, -0.3572828471660614, 1.2420835494995117, -1.769895076751709, -1.0827425718307495, 0.30566829442977905, 1.540809988975525, -0.07196085155010223, 0.4759819507598877, 0.28496572375297546, 0.02243814431130886, -0.20895349979400635, 0.8261560797691345, -0.9628385305404663, 0.1407274603843689, 0.3003188371658325, 1.888177514076233, -0.6030192971229553, -1.7474466562271118, 0.02124827727675438, 0.6512080430984497, 0.26664137840270996, 0.09370356053113937, 0.20403029024600983, 2.7846944332122803, -0.5112408995628357, 0.32372933626174927, 0.09213114529848099, 0.03863262012600899, -0.02523493394255638, 0.9114649891853333, 0.5142284035682678, -0.8858271837234497, 0.7046908140182495, -2.361408233642578, -0.2638222575187683, 1.6331570148468018, 0.30448034405708313, -1.6454356908798218, 0.498489648103714, 0.1080859825015068, -2.399172782897949, 0.9907580614089966, -0.8191878795623779, -0.8788199424743652, -0.5881022810935974, 0.4165700078010559, 0.40571215748786926, -0.06901006400585175, -0.1126224622130394, -0.012331278994679451, -0.7530018091201782, 0.07544336467981339, 1.320078730583191, -0.5037168264389038, -1.024564266204834, -0.31553035974502563, -1.297348141670227, 0.19305652379989624], [-1.6195191144943237, 1.1176090240478516, 0.3045276999473572, 0.73186856508255, 0.7556677460670471, 0.5253662467002869, -0.26308712363243103, -1.4308940172195435, 1.0474971532821655, -1.481515884399414, -1.0730479955673218, 0.3642328679561615, 1.4988425970077515, -0.21995952725410461, 1.3313522338867188, 1.8695495128631592, -0.30731409788131714, 0.840011715888977, -1.7553796768188477, -0.32412734627723694, -0.2650485932826996, -0.13088752329349518, 2.2719461917877197, -1.3179007768630981, 0.8368424773216248, -1.5743900537490845, -0.8176091909408569, 0.10020177066326141, -1.3409204483032227, 0.9148527979850769, 1.2478054761886597, 1.3596150875091553, -0.45887672901153564, 0.5915820598602295, 1.7082542181015015, -0.42130276560783386, -2.3683764934539795, -1.4563120603561401, 0.6176238656044006, -0.0002397881617071107, -1.1440776586532593, 1.3725539445877075, -1.4936126470565796, 1.7543281316757202, 0.5854710340499878, 0.8126130700111389, -1.3074606657028198, 0.9110721349716187, 0.055697206407785416, -2.1834042072296143, -0.6970726251602173, 0.13331253826618195, -0.3768630921840668, -0.9656357765197754, 0.26831722259521484, -0.3026043772697449, 0.8809223175048828, 0.6969940662384033, -0.6844940781593323, 0.012275384739041328, -0.9336890578269958, -0.48163819313049316, -1.0260971784591675, -0.6864296793937683], [-0.49371543526649475, 0.7491885423660278, 0.881344735622406, -1.3133528232574463, -0.9690058827400208, 0.25926846265792847, -0.3163173794746399, 1.3009541034698486, -0.9445716142654419, 0.2209084928035736, -0.07345806062221527, 0.22522905468940735, 1.8905906677246094, -0.7349676489830017, -0.2614421248435974, 1.0413907766342163, 0.17159868776798248, -0.06384529918432236, 0.7435176372528076, -1.435563087463379, -1.3381156921386719, 1.2353278398513794, -2.555283546447754, 0.7877090573310852, -1.6473026275634766, -1.25342857837677, -1.7041453123092651, 0.4319339692592621, -1.7110651731491089, 0.9272358417510986, 1.873735785484314, 0.3316706418991089, -2.68823504447937, 0.6083597540855408, 0.04646725580096245, -1.138871192932129, 0.04095926880836487, 0.2785104811191559, 1.3464949131011963, 0.33297133445739746, 0.5436718463897705, 0.9012959599494934, -1.3103586435317993, 1.9104304313659668, -0.5283154249191284, -1.7565547227859497, -0.2315211296081543, 0.7806328535079956, 0.049871888011693954, -0.23936495184898376, 0.2433650642633438, 0.0895705595612526, -0.6217085123062134, -0.5527319312095642, 0.56305330991745, 0.36924684047698975, -0.17499908804893494, 1.6991028785705566, 0.3620429039001465, -0.6087568402290344, 1.714593529701233, 2.3678088188171387, 0.23537012934684753, 0.5330712199211121], [-1.593173623085022, 0.16562536358833313, 0.556300699710846, -0.9007005095481873, -0.8136865496635437, 1.4342272281646729, -0.24211491644382477, -0.899931013584137, -0.29417556524276733, -1.329875111579895, 0.2807789444923401, -0.7712030410766602, -0.3433745801448822, -0.380985289812088, -0.10071726143360138, 1.285901665687561, 0.4946194291114807, -0.4650958180427551, 0.6246490478515625, 0.730431318283081, 0.1392775923013687, 0.0891331136226654, -0.09622815251350403, -0.5547558665275574, -0.7091337442398071, -0.5081645846366882, -1.131727695465088, 1.1795870065689087, 1.205184817314148, 1.4315615892410278, -0.5831477046012878, 2.1300811767578125, -1.8458350896835327, 0.06934353709220886, -2.0839858055114746, -1.7373831272125244, -0.36742809414863586, 0.35200947523117065, -0.9817732572555542, 1.0195614099502563, 0.35419848561286926, 0.2562890350818634, -1.3508601188659668, -1.03656005859375, 0.6248981952667236, 1.3717271089553833, -1.015385627746582, 0.1630788892507553, 0.06811325252056122, -0.5928520560264587, 0.7661135792732239, -1.0438426733016968, 0.3106723725795746, -0.8953496217727661, -0.7821157574653625, -1.291000485420227, -0.8484300971031189, 0.5332871079444885, -0.23574694991111755, -0.7059083580970764, 0.4382892847061157, -0.9801234006881714, -0.3670782744884491, -2.519554376602173], [-0.7612433433532715, -0.04162370786070824, -0.7550448179244995, 0.4524303674697876, 1.0132038593292236, -0.28847014904022217, 1.0123393535614014, 0.8499543070793152, 1.0767300128936768, 0.3683648705482483, 0.1750197559595108, 0.7497605681419373, -0.4551176130771637, -0.24292024970054626, 2.1825225353240967, -0.9694066643714905, -0.3873652517795563, 0.18521878123283386, -1.4184261560440063, -0.7524237632751465, 1.1861311197280884, 2.2455790042877197, 0.6395629048347473, 0.2736235558986664, 0.5329625010490417, -1.203050971031189, -0.31270596385002136, -0.8865986466407776, -0.168679341673851, -0.6090999841690063, -0.07588376849889755, -0.5093075633049011, -1.1021250486373901, -1.1093472242355347, 2.148747205734253, -0.6471138000488281, -0.716455340385437, -1.5390647649765015, -1.3133280277252197, 0.8590308427810669, 0.9266589879989624, 2.464961528778076, 1.562838077545166, 1.3000853061676025, 0.47317957878112793, -0.7576568126678467, 0.5917749404907227, 0.6741386651992798, -0.887780487537384, 0.40182989835739136, 2.378906726837158, 0.45875123143196106, -1.2137523889541626, -0.5336735844612122, 0.15241071581840515, -0.08355901390314102, -0.6457423567771912, -0.3457968235015869, 0.5739554762840271, -0.07495362311601639, -2.285674810409546, 0.12176574021577835, 3.017533540725708, -0.04364113137125969], [0.17238590121269226, 0.6796131730079651, -0.8601725101470947, -1.2970391511917114, 1.4700002670288086, -0.10234861075878143, 0.4258193075656891, -1.19979727268219, 0.6637035608291626, -1.1283916234970093, 0.5889641046524048, 0.8613385558128357, -1.7774107456207275, 0.053068939596414566, -0.15976086258888245, 2.225595712661743, -0.8363420367240906, -0.5308890342712402, -0.31249478459358215, -0.2815876603126526, -0.4227628707885742, -1.0677604675292969, -0.9174379706382751, 1.030045509338379, -0.7089900970458984, -0.15770219266414642, 0.44907134771347046, -0.6804105043411255, 0.3988723158836365, 1.8992562294006348, -0.0033599534071981907, 1.8202229738235474, 0.38943764567375183, 0.37702974677085876, 1.0679676532745361, -1.1683903932571411, -1.0500637292861938, 0.9644362926483154, -0.6027464270591736, -0.387736052274704, -0.3941480219364166, -0.7599577903747559, 1.4456734657287598, 1.1110788583755493, -1.6904258728027344, -0.5883588790893555, -0.37179437279701233, -0.06050301343202591, -0.8017759323120117, -1.2932791709899902, 1.093412160873413, -0.13140228390693665, -0.09764506667852402, -1.4013645648956299, -2.265475273132324, 1.3573639392852783, -1.2142858505249023, -0.8733189702033997, 0.6573278903961182, -0.5327040553092957, 0.06327569484710693, 0.042950160801410675, 0.8821061253547668, -0.19251768290996552], [-0.03392772749066353, 0.24529199302196503, 0.8201204538345337, -1.804636836051941, 0.402790367603302, 1.8306164741516113, 0.14413593709468842, -0.7033854126930237, -0.6004555821418762, 1.8913109302520752, -0.4743013083934784, 2.540938138961792, -1.1403628587722778, 1.3484128713607788, 0.077278271317482, 2.495530128479004, -0.44401413202285767, -0.04141053557395935, -0.6029425263404846, -0.3970557153224945, -0.6541447639465332, 0.6504684686660767, 0.6075919270515442, -2.93923020362854, 0.5125151872634888, -0.36987581849098206, 0.0708606094121933, -1.559167504310608, -1.1390876770019531, -1.4513782262802124, -0.764371395111084, 0.7746532559394836, 1.2543314695358276, 1.5739494562149048, 1.2677100896835327, -0.4527744650840759, -1.110700249671936, 1.5987939834594727, 0.5899741649627686, 0.16205953061580658, -0.6255231499671936, 1.1591718196868896, -0.6186549067497253, 0.6858085989952087, -0.7431904077529907, 0.27064457535743713, 1.1692756414413452, -0.04952235892415047, -0.35863959789276123, 0.562380850315094, 0.4367942214012146, -1.660706639289856, -0.2460053414106369, 1.0768741369247437, 0.3670922815799713, 0.9637751579284668, -1.1408822536468506, 0.9021348357200623, 1.132575273513794, -1.1898722648620605, 0.020263679325580597, 1.8421108722686768, 0.39026299118995667, 1.2868059873580933], [0.5202174186706543, 0.9234699606895447, 0.11435915529727936, 0.9869865775108337, 1.5853453874588013, 0.9127761721611023, 0.4098837971687317, -0.4856715798377991, -0.32553112506866455, 1.568426251411438, 0.05434364452958107, 0.8873607516288757, 0.33437055349349976, 0.26690250635147095, -0.03943980485200882, 0.5076553821563721, -1.9188152551651, -2.1057932376861572, -0.1290975660085678, -1.3571127653121948, -1.133697748184204, 0.08970048278570175, -0.9879499673843384, -0.7519435286521912, -0.12229619920253754, -1.1368874311447144, -0.891620397567749, 0.7282794117927551, 0.9283187985420227, -0.3843282461166382, 0.39916932582855225, -0.8522323369979858, -1.55496346950531, 0.7182067632675171, -0.9439762830734253, -0.28079232573509216, 0.3905712366104126, -0.7865955233573914, 1.3209142684936523, 0.5430042743682861, -0.16830280423164368, 0.24968306720256805, -0.8074750304222107, -1.090444564819336, 1.831146478652954, 1.4465709924697876, 0.5002844929695129, -0.5351366996765137, 0.016492396593093872, -0.44594109058380127, 1.6852617263793945, -0.7655326128005981, -0.4119795560836792, -0.7734013795852661, -0.692462682723999, -0.46377646923065186, -1.3729515075683594, -0.7640754580497742, -0.15300799906253815, 1.573869228363037, 1.7000168561935425, 0.22480523586273193, -0.05152782052755356, 1.5104584693908691], [-1.57405424118042, 1.701775074005127, 1.9831689596176147, 0.5807918310165405, 0.4361829459667206, -1.271862506866455, 1.089311957359314, 0.29815080761909485, 0.4381912052631378, 1.4554373025894165, -0.42340317368507385, -0.5929558873176575, -0.3018296957015991, -1.9529746770858765, -0.3070092499256134, 0.04945647716522217, -0.007595240604132414, 1.4309499263763428, -0.7527310848236084, 0.3208865523338318, -2.1202926635742188, 0.7444701790809631, 1.122935175895691, 0.40690648555755615, -0.6348193883895874, 0.5213746428489685, -1.0652406215667725, -0.5938952565193176, 3.23050856590271, 0.9957002997398376, 0.7653993964195251, 0.36756619811058044, -0.6551340222358704, -0.35069069266319275, -0.43192538619041443, 1.4491099119186401, 1.804740071296692, 0.2037135511636734, 0.7749464511871338, -0.42302194237709045, 0.2141955941915512, 1.1365586519241333, 0.933791995048523, 0.28310465812683105, 1.2092171907424927, 0.4335152804851532, 1.7082713842391968, -0.044066172093153, 0.09151522815227509, 2.132154941558838, -0.9998602271080017, 0.34314754605293274, -1.51410973072052, 1.1294543743133545, -1.3894506692886353, 0.15333078801631927, 0.4520946145057678, 0.29765164852142334, 0.035132646560668945, 0.3861789405345917, -0.3814839720726013, -0.21682055294513702, 0.36932137608528137, -0.01227621641010046], [0.8392583131790161, -0.9932416081428528, -2.0686697959899902, 0.34380611777305603, -0.34876003861427307, 2.932576894760132, -0.083565354347229, 0.5813689827919006, -1.0641642808914185, 0.07847519218921661, 0.46102970838546753, 0.4573330283164978, -0.5608150959014893, -0.35914933681488037, 0.22596794366836548, 0.42490455508232117, -0.034932490438222885, -0.44319358468055725, -0.33405742049217224, -0.8742695450782776, -1.0418829917907715, 0.2459505796432495, 1.9187281131744385, -0.9618164300918579, 0.08295950293540955, -0.8356189131736755, -2.445523738861084, -1.1754653453826904, -0.2106623351573944, 1.2263360023498535, -1.3157527446746826, -0.8463759422302246, -0.17542077600955963, -1.7814806699752808, -3.0999364852905273, 0.9365456700325012, 1.6417193412780762, 1.8165464401245117, 0.5644477009773254, 1.6910998821258545, 0.8611050844192505, 0.8156708478927612, -1.0980925559997559, 0.4212113618850708, 0.8217452764511108, 0.700019896030426, -2.054760217666626, 0.1610557585954666, -1.4825489521026611, 1.740493655204773, 0.7601991295814514, 0.2858092188835144, 2.0113449096679688, -0.6833046674728394, -2.530924081802368, 0.41098326444625854, -1.5116617679595947, 0.2801263630390167, -1.1910350322723389, 0.4450426399707794, -0.4012161195278168, -0.035477329045534134, 0.3374583423137665, -0.9873823523521423], [0.42939281463623047, -0.2401459515094757, -0.34052473306655884, 0.5180439949035645, -0.07036988437175751, -0.4419389069080353, -1.1647354364395142, -0.7215591073036194, -0.17045138776302338, -1.0447808504104614, 1.745990514755249, -0.09303584694862366, -0.1291533261537552, 0.013967587612569332, 0.12610404193401337, 0.06358982622623444, 1.300401210784912, 1.103867530822754, -0.8189914226531982, 0.1444738209247589, -0.5948743224143982, 0.22487269341945648, -0.45931366086006165, 1.0372374057769775, -0.8761687874794006, 2.083578109741211, 0.946392834186554, 0.8193901777267456, -0.7521002292633057, 2.64396333694458, 0.2407991737127304, -1.0311039686203003, -0.6419506669044495, 0.474673867225647, -1.149118185043335, 0.46815207600593567, -1.0593031644821167, -0.4390351474285126, 1.192120909690857, -2.3170266151428223, -0.7907028198242188, 0.22690297663211823, -1.395241379737854, -0.8322435021400452, 1.0427569150924683, -0.2029496133327484, 1.9654632806777954, 2.0691440105438232, -0.4733448028564453, 0.18254539370536804, 0.42585086822509766, -0.9460352659225464, -1.5107709169387817, -1.3674236536026, -0.38955622911453247, -0.9849151968955994, -1.23367440700531, 0.2012900710105896, -0.031088534742593765, -0.6380552649497986, 0.924292802810669, 1.3988794088363647, 1.2711745500564575, -1.0372276306152344], [-0.5463982820510864, 0.07621902972459793, 0.24225816130638123, 0.7658042311668396, -0.004386011976748705, -0.4283565282821655, 0.26823413372039795, 0.5746755599975586, -1.6425583362579346, -0.04166988283395767, -0.8969465494155884, -0.3500778377056122, -0.04187659174203873, -1.3582597970962524, 1.6588331460952759, 0.9180310368537903, 1.2629930973052979, -1.5214507579803467, -0.08521606773138046, -2.0349133014678955, 0.9418061971664429, 0.42178988456726074, -1.5941771268844604, 0.7144536375999451, 0.3051603436470032, 0.6581330299377441, -0.08466807752847672, -0.16109609603881836, -1.8020089864730835, -0.7582404017448425, 0.33707740902900696, -0.20178493857383728, -0.20420454442501068, 0.9347291588783264, -0.613447904586792, 2.1215784549713135, -0.10884113609790802, 1.601407527923584, 0.4449167549610138, -1.4345743656158447, 0.17714741826057434, 0.6532540321350098, -3.1972920894622803, 0.5657179355621338, -0.46173444390296936, -1.4287867546081543, -0.10633671283721924, -1.0864235162734985, -0.06413105875253677, 0.0373239740729332, -0.6756035685539246, -0.15895332396030426, -1.3309292793273926, 0.320482075214386, -0.20836812257766724, -1.408359408378601, 0.45785409212112427, -0.5275207161903381, -0.2837797701358795, -0.536308765411377, 0.33647140860557556, 1.5562266111373901, -0.08037105202674866, -0.14766012132167816], [-0.503588080406189, 0.5277851819992065, 1.6839760541915894, -0.8746175765991211, 0.5361838936805725, -0.6486783623695374, -0.587013304233551, 1.6404434442520142, 0.9170085191726685, 1.1045947074890137, 0.020768718793988228, -0.22076109051704407, 1.5371215343475342, -0.16829024255275726, 1.6966350078582764, 0.46183091402053833, -0.2698022127151489, 0.02675178274512291, 0.5061941742897034, -1.0540851354599, 0.8574246168136597, 0.7492170929908752, -0.255436509847641, 0.17510217428207397, -0.6504946351051331, 0.24791760742664337, -0.2894532084465027, 0.08174039423465729, -0.599576473236084, 0.636520266532898, -0.5036954879760742, -1.3128806352615356, 0.4481607973575592, 1.7817611694335938, -1.2561203241348267, -0.239519402384758, -0.5268909335136414, -0.4295503795146942, -0.8138374090194702, -0.6117813587188721, -1.938787579536438, 1.3035215139389038, 0.9384351372718811, 1.511408805847168, 1.5299365520477295, 0.13004791736602783, -0.3043975532054901, -0.8325676918029785, 0.19650840759277344, 1.0203471183776855, -1.2114677429199219, 1.3136307001113892, 0.37685805559158325, -1.8863426446914673, 0.28978225588798523, 0.8808176517486572, 0.5767344832420349, 0.6488307118415833, -0.9008892178535461, -0.42055800557136536, -0.7249534130096436, 1.6947485208511353, 0.4616905152797699, 0.9713141918182373], [-0.39037877321243286, -0.6324970722198486, 0.16649268567562103, 0.032712049782276154, -1.3532452583312988, 0.16197343170642853, -0.5150298476219177, 0.10904187709093094, 0.9963874220848083, -1.4732898473739624, -1.4520491361618042, 0.2555205225944519, 0.17784881591796875, -2.127631187438965, -0.2470608353614807, -0.45806410908699036, 0.7795909643173218, -0.7815989255905151, 1.4271389245986938, 0.33157163858413696, -0.3658972978591919, 0.9620028138160706, 1.0223875045776367, 0.3843362033367157, 2.478759288787842, -0.6636650562286377, -0.665738046169281, -1.6935395002365112, -0.5378678441047668, 0.34883055090904236, 0.9038659930229187, 0.04443356394767761, -0.8567112684249878, -0.3189143240451813, 0.7218998074531555, -2.509922742843628, -0.4744333326816559, -1.0362064838409424, -1.1320748329162598, -0.5032076835632324, -1.196161150932312, -1.5161850452423096, 0.26258429884910583, 1.3659840822219849, -0.17929048836231232, -0.09904724359512329, 2.0316967964172363, 0.38118505477905273, -1.4359381198883057, -0.48144975304603577, -0.20212677121162415, 0.8252358436584473, -0.7280987501144409, -0.3538206219673157, 0.9294382929801941, -0.5257709622383118, 2.0928611755371094, 1.029189944267273, -1.276463508605957, 0.8898499608039856, 0.22638732194900513, 0.2507297992706299, 0.6199710369110107, 0.5896958708763123], [-0.30835819244384766, -0.6878288984298706, 1.7879024744033813, -0.08388033509254456, 1.4229629039764404, -1.3109997510910034, 0.8336414098739624, -0.8092437386512756, -0.5438512563705444, 1.0616250038146973, 0.842196524143219, -0.5419526696205139, 0.6725797653198242, 0.435413658618927, -0.2534644305706024, -0.6755650043487549, -0.11657930165529251, -0.8191134333610535, -0.47921326756477356, -1.9843945503234863, -0.5201743841171265, 1.3302370309829712, 1.772046685218811, 0.652789831161499, -0.0717548355460167, -0.5818666815757751, 1.1205072402954102, 0.4952450692653656, -0.018060855567455292, 0.7183488011360168, 1.1288502216339111, -0.1644294112920761, 0.03130044415593147, -0.22747153043746948, 2.1781232357025146, 0.9463343620300293, 0.9972085356712341, 0.8729557394981384, -1.4827104806900024, 0.5842435359954834, -0.2745760679244995, -0.7156583070755005, -1.3059815168380737, 0.8387443423271179, 0.6138898730278015, -0.3798424303531647, 1.3349430561065674, -1.273412823677063, 2.0707168579101562, 1.1965614557266235, 0.36280885338783264, -0.5535997152328491, -1.3416929244995117, -1.730027198791504, 0.4499850869178772, 0.1958276629447937, 0.023098289966583252, 0.3119058310985565, -0.3638128638267517, -1.5485522747039795, -1.0470246076583862, -0.09249038249254227, 0.47626665234565735, -0.8344274163246155], [-0.10841965675354004, -1.765089750289917, -0.12247660011053085, -0.24283760786056519, -1.7581498622894287, -0.40837714076042175, -0.5052257776260376, -1.2648037672042847, 2.4499852657318115, 0.622804582118988, -1.078308343887329, 1.2067281007766724, 0.025715339928865433, 0.9194490909576416, -0.42240196466445923, 0.11662391573190689, 1.6966804265975952, 0.22111117839813232, -0.08073682337999344, -1.2237319946289062, -0.020510461181402206, -0.7425699234008789, 0.14168938994407654, -0.7348553538322449, 0.7709089517593384, 0.8463018536567688, 0.3504020869731903, -0.37652304768562317, -0.30409660935401917, -0.5090452432632446, -1.1452490091323853, 0.7785676717758179, 0.6526012420654297, 0.3499484956264496, 0.12295673042535782, 1.2836370468139648, -0.7705685496330261, 1.7259680032730103, -0.3053576648235321, -0.13837921619415283, 0.34211182594299316, 0.413883775472641, -1.4590580463409424, 0.5748220086097717, 1.2174739837646484, 0.3217843174934387, -0.22993971407413483, 0.8676524758338928, -0.5230629444122314, 0.31580984592437744, -0.23064622282981873, -0.4653824269771576, -1.1610280275344849, -0.215157613158226, -0.027897967025637627, 0.10228051245212555, 0.17387577891349792, 1.4291901588439941, 0.43438565731048584, -0.49434298276901245, 0.39399123191833496, -0.21123473346233368, -0.7913910746574402, 0.14800523221492767], [-0.3931446969509125, 1.2581843137741089, 1.058281421661377, 0.5302109718322754, 0.9819235801696777, -0.6193515658378601, 0.6023041009902954, -1.385423183441162, 0.5452916622161865, -1.80399489402771, -0.3902007043361664, 0.5656102895736694, 0.4770938754081726, -1.3877403736114502, 1.5408365726470947, 0.3534306287765503, -0.001816107309423387, 0.5070351958274841, -0.6858408451080322, -0.07772937417030334, -1.7589553594589233, 0.9540843963623047, -0.7855640053749084, -0.0217859148979187, 0.22541728615760803, -1.8843958377838135, 1.395396113395691, 0.12566132843494415, -0.8273608088493347, 1.298990249633789, -0.8924235701560974, -0.4249512851238251, -0.7487109899520874, 0.011143846437335014, -0.11928914487361908, -0.2721827030181885, 1.4064825773239136, 1.1891319751739502, -0.8872398138046265, 1.0516241788864136, -2.323613166809082, -0.9807252883911133, -0.7703086733818054, 2.326385498046875, -0.9256243705749512, -0.14711129665374756, -0.7329265475273132, -0.8156248927116394, 0.17576806247234344, 1.770373821258545, 0.6568955183029175, 0.33773520588874817, -0.1439555287361145, 0.038747143000364304, 1.072128415107727, -0.8764720559120178, -0.7425422072410583, 0.8405311703681946, -0.056103482842445374, 1.8154597282409668, 0.25645819306373596, -0.7109804749488831, 0.014106672257184982, -2.067718267440796], [2.082267999649048, 0.858330249786377, -0.07193908095359802, -2.0997190475463867, 1.3882172107696533, -0.17456510663032532, 0.5734438300132751, 0.025287723168730736, 2.0309910774230957, 1.2262496948242188, 0.9741213321685791, 0.27222704887390137, 2.1394906044006348, 0.39200738072395325, -0.8258235454559326, 0.7870752215385437, 0.41878971457481384, -0.2343607097864151, 0.4645783007144928, 1.256947636604309, 0.8655239939689636, 1.5300763845443726, -0.4384504556655884, 0.8534730672836304, 1.0979137420654297, 0.8601534366607666, -0.9881248474121094, 0.0876196026802063, -0.5255884528160095, -1.262637734413147, -0.20846182107925415, -1.8192299604415894, 0.7630674242973328, 0.16691581904888153, 0.8002722859382629, -1.4690070152282715, 1.2545198202133179, -0.7148729562759399, 0.8665266633033752, 0.7812029719352722, -1.056991696357727, 0.3092515170574188, 1.1656746864318848, 0.4475451707839966, 0.20826564729213715, -0.24165157973766327, 0.26492518186569214, 0.08712535351514816, 0.1517663598060608, 0.9586849212646484, 0.8407571911811829, -1.2581664323806763, -0.034022048115730286, 0.9395610690116882, 1.1476119756698608, -0.2591765224933624, -0.7478753328323364, -0.5365957617759705, -0.6241365671157837, -1.0085567235946655, -0.22520610690116882, -1.8748713731765747, -2.3367810249328613, 1.8332678079605103], [-0.45308345556259155, 1.6642394065856934, -0.6887414455413818, 2.322329044342041, -0.4332207143306732, -0.7085031867027283, 0.8907425403594971, -1.8203572034835815, 2.258779525756836, 0.804442286491394, 1.015138864517212, 0.10300444811582565, -0.8454105854034424, 1.1882257461547852, -0.3929370641708374, 0.9444635510444641, 0.7593513131141663, 0.19936908781528473, 1.3531873226165771, -1.0838873386383057, -1.6123768091201782, -0.6372601389884949, -0.3321966826915741, 0.2462194859981537, 0.35829731822013855, 0.03576415777206421, 0.5163002014160156, -1.3398020267486572, -1.156175136566162, -0.8761732578277588, -0.5912894606590271, -0.7633717656135559, 0.2526546120643616, -0.8624985218048096, 0.9913026690483093, 0.7697427868843079, -0.11502871662378311, 0.7555876970291138, -0.07720854133367538, -0.07724806666374207, -1.1272822618484497, -0.7554734349250793, -0.7910692691802979, -0.8326535820960999, 1.1702340841293335, 0.08076788485050201, 1.46923828125, 0.6939085721969604, -0.26508864760398865, 0.8151370882987976, 0.6927616596221924, -0.603165864944458, -0.7440248727798462, 0.3123106360435486, -0.7386162281036377, -0.8174549341201782, 2.0793564319610596, -1.4537798166275024, -2.284832715988159, 0.36642104387283325, 0.4498002529144287, 0.6925323009490967, 0.517474353313446, 0.15413036942481995], [0.8084337711334229, 1.784633755683899, 0.7061118483543396, 0.16176441311836243, -0.1565326601266861, 1.7433373928070068, 1.9315478801727295, 0.5792763829231262, 0.25201526284217834, 0.27958908677101135, 0.7726029753684998, 0.9084389805793762, -0.5485954880714417, 0.8344293236732483, -0.12028403580188751, -0.4773377776145935, -0.15478786826133728, 0.34917840361595154, -0.14038677513599396, 1.4934998750686646, -0.5084203481674194, -1.08155357837677, 0.20400768518447876, 0.7749447226524353, 0.5441543459892273, 0.6317431926727295, 0.3296900689601898, 0.6520863175392151, 0.34967145323753357, 0.33352431654930115, -0.23344223201274872, -1.142045497894287, 0.9542754888534546, 0.9172109365463257, 0.2654353082180023, 0.19587282836437225, 1.615044355392456, 0.7063331604003906, 0.5284212231636047, -0.6870107650756836, -1.1170921325683594, -0.9755219221115112, -1.0553908348083496, 0.4109364151954651, 0.025155380368232727, -0.6810474991798401, 0.8451069593429565, 0.13255879282951355, -1.0463515520095825, -2.7694451808929443, -0.8345605731010437, 0.6595954298973083, -0.06008584424853325, 0.4077202379703522, 0.6473416686058044, -1.0742554664611816, 0.005242036655545235, -0.3580818772315979, 0.0832766517996788, -0.4679874777793884, -1.5676913261413574, 0.542968213558197, 0.7387866973876953, 0.503512442111969], [0.02896612137556076, 0.699335515499115, 0.3268873393535614, -0.6647475361824036, 1.5088396072387695, -1.2561182975769043, -0.7093453407287598, 0.5701075196266174, 1.067931890487671, -0.3539685904979706, 0.5116577744483948, -0.8980122804641724, -0.47613295912742615, 1.2135286331176758, -0.11426158994436264, 0.17984922230243683, -0.960712730884552, -1.8490952253341675, 0.5720109939575195, 0.14066872000694275, -0.8990238308906555, 0.01037220936268568, 0.7828263640403748, 1.313798427581787, 2.6081268787384033, 0.039442647248506546, -2.298719882965088, -0.4200166165828705, -0.19549942016601562, 0.04718407243490219, 0.6245556473731995, -1.8408212661743164, -1.961812138557434, 0.9383783340454102, -0.1318901777267456, -0.6172543168067932, -0.04022519662976265, 0.4869404733181, 0.1932721585035324, -1.435060739517212, 1.0503243207931519, 1.2919081449508667, -0.991468071937561, -0.5702850222587585, -0.25546929240226746, 1.4067102670669556, 0.6284612417221069, -0.7378645539283752, -0.06614695489406586, -0.38334187865257263, 0.43910032510757446, 0.3321574330329895, -0.5782370567321777, 2.251725673675537, -0.34568914771080017, 0.1012134775519371, -0.1581660360097885, -0.10271662473678589, -0.9267674088478088, 0.4087921679019928, 0.7422495484352112, 0.3645259141921997, -0.5982562303543091, -0.8577153086662292], [-0.2476217895746231, 0.01516188494861126, 1.9193373918533325, 0.59181809425354, 1.38358736038208, 0.6392390131950378, -1.2594237327575684, 2.7842071056365967, -0.7177549004554749, 0.3073573410511017, 0.11058352887630463, 0.6933440566062927, 0.8130166530609131, 0.9349952936172485, 0.39263445138931274, -0.4688117206096649, 1.3582520484924316, -1.3606208562850952, 0.9940125346183777, 1.1599326133728027, 0.4169479310512543, 0.5490949749946594, 1.0282199382781982, -0.6810705065727234, 0.16493605077266693, 1.280226707458496, -2.0507752895355225, -0.8293114304542542, -0.8034275770187378, -0.22142744064331055, 0.6272461414337158, -0.30323511362075806, 1.66946542263031, -1.6120485067367554, -1.2656618356704712, 0.5374600291252136, 0.6729663610458374, 0.698513388633728, 0.045589227229356766, 0.33483538031578064, -0.49865373969078064, 0.6835736632347107, -0.32069292664527893, -1.0592764616012573, -1.7271411418914795, 1.1278959512710571, -0.37939366698265076, 0.7032801508903503, -0.3495168387889862, 0.8044744729995728, -0.002158200368285179, 0.2200430929660797, 0.6613910794258118, -2.0921120643615723, -0.7808287739753723, 0.04970309138298035, 0.9808487296104431, -0.11051458865404129, -0.46786895394325256, -0.37953054904937744, -0.23922847211360931, 0.35328203439712524, 1.070977807044983, -0.39550745487213135], [-0.733092188835144, 0.35578683018684387, 0.8463426828384399, 0.7144092321395874, -1.0635488033294678, -0.46144217252731323, 1.1261922121047974, 0.2869626581668854, 0.6479625701904297, -2.093137502670288, -0.5433927178382874, -1.2906144857406616, -1.9186172485351562, -0.016692858189344406, -1.0284479856491089, 0.6521150469779968, 0.2402588576078415, 0.6615021228790283, 1.9636932611465454, -0.26371070742607117, -0.7723408937454224, 0.38182032108306885, 1.1666728258132935, 0.0513322576880455, 0.32240158319473267, -0.12822361290454865, 0.01766718365252018, 0.8118864297866821, -0.9942343831062317, -0.25067922472953796, 0.10240351408720016, 0.42834165692329407, 0.3516278564929962, -0.7810606956481934, 0.40862420201301575, -0.24979133903980255, -0.2070085108280182, 0.06212547793984413, -0.7303385138511658, -1.68308424949646, 0.6102889180183411, 0.07287902384996414, -0.839867115020752, 0.30676063895225525, -1.2374480962753296, 0.5849100947380066, 0.12117090076208115, -0.7914044857025146, 0.17480212450027466, -0.6341189742088318, 0.38597413897514343, 0.24578727781772614, 1.6770836114883423, -0.15893325209617615, -0.7858151793479919, 0.1643366664648056, -0.4437474012374878, 0.9706907868385315, 0.5512655377388, -0.08184509724378586, -0.8874648213386536, 0.7990871071815491, 1.2593255043029785, -1.1691209077835083], [0.5679841637611389, -0.04304379224777222, 0.644451916217804, 0.6607295274734497, -0.07703225314617157, -0.6179763078689575, 0.21023795008659363, -0.6809910535812378, 0.8083038330078125, -0.029816342517733574, -1.3062578439712524, 0.9857175946235657, 0.10016770660877228, 0.46218135952949524, -1.4821922779083252, 0.698104202747345, -0.9521995186805725, 0.3185977637767792, -0.1982194185256958, -0.899392306804657, 1.921905755996704, -0.12557287514209747, -0.7906050682067871, -0.4016670882701874, 0.2298998236656189, 0.23504656553268433, -0.013977707363665104, -1.3170397281646729, -0.25027766823768616, -0.07638765126466751, 0.8094602823257446, -0.10030148923397064, -0.8969361782073975, -1.8933320045471191, -0.9140294790267944, 0.7361956238746643, 0.4201110303401947, -0.09835124015808105, 1.2695732116699219, 0.16546951234340668, 0.7806777358055115, -0.7085703015327454, -0.022231819108128548, -0.41886669397354126, -0.13483302295207977, -1.7804467678070068, -0.07467693090438843, -1.0660195350646973, -0.7060117721557617, 0.10560621321201324, 0.17081846296787262, 0.22829695045948029, 0.2818678617477417, -0.3786453306674957, 0.2691839039325714, 0.7794956564903259, -1.9868053197860718, 1.6321386098861694, 0.0009639447671361268, 0.49643629789352417, -0.0014234452974051237, -0.4668275713920593, -1.1530430316925049, 0.7273685336112976], [-0.9838582277297974, -0.13759151101112366, -0.4107738435268402, -0.25989753007888794, -0.5351154804229736, -0.14215686917304993, -0.3158416748046875, -0.7436663508415222, 1.602972149848938, -0.3451172411441803, 1.0331430435180664, 0.16707493364810944, 0.12043342739343643, 0.5359392166137695, -1.0798345804214478, 1.0620083808898926, 0.46759968996047974, -0.8490248918533325, -1.1415846347808838, -0.5034039616584778, -1.594997525215149, -1.3572850227355957, -1.8803768157958984, 1.8418588638305664, -0.06813903898000717, 0.6471922397613525, 0.5832110047340393, -0.898140013217926, -0.9480543732643127, -0.3502170443534851, 1.6976486444473267, -2.659428358078003, 0.4678785800933838, 1.7106105089187622, 0.23518158495426178, -1.4285467863082886, -0.6873496770858765, 0.2970384955406189, -0.9848953485488892, -0.8766247630119324, 0.5011122226715088, -0.0727236419916153, -0.757193386554718, -0.145046666264534, 0.5785553455352783, -0.3662991225719452, 0.612846314907074, -0.6148044466972351, -0.2042178064584732, 0.2419346570968628, -0.22380009293556213, 0.7874117493629456, 1.5526803731918335, -0.3685305118560791, -0.26962944865226746, 0.9853240251541138, 0.4412165880203247, 0.269252747297287, -0.2616829574108124, -0.7538784742355347, -0.6639911532402039, 1.0127137899398804, 1.557274580001831, -0.7681196928024292], [-0.8380181789398193, -1.2156902551651, 1.3203119039535522, -0.20180755853652954, -0.22987359762191772, -2.1516520977020264, -1.553787350654602, 0.3377111554145813, -0.6617470979690552, 0.4317471981048584, 0.37566426396369934, 0.4438917934894562, 0.2636164426803589, 0.5965536832809448, -0.3917015790939331, 0.8879421949386597, -1.0164895057678223, -0.9327067136764526, -1.344048261642456, 1.2663151025772095, 0.7167462110519409, -0.4218597710132599, -1.5784565210342407, -0.7804738283157349, -1.2392442226409912, -0.19291891157627106, -0.32041504979133606, -0.3504536747932434, -1.5466158390045166, 0.492458313703537, 0.4770757257938385, -1.215664029121399, 0.11221455037593842, -1.0321670770645142, -0.3029647767543793, 0.6379568576812744, 1.5143353939056396, -0.9643812775611877, -1.0327019691467285, 0.19610470533370972, -1.8382220268249512, 0.046832140535116196, -1.1998355388641357, -0.9644256234169006, 0.5245112180709839, 0.4429709017276764, -0.23992279171943665, 0.46339255571365356, -0.05447029322385788, -0.5050913095474243, -1.721691608428955, 0.46643730998039246, 0.7453566789627075, 0.16886138916015625, 1.4155150651931763, -0.11875555664300919, -1.196233868598938, 1.5307648181915283, 0.7102023959159851, 0.5830044746398926, 0.42549851536750793, -0.12601812183856964, -0.8559879660606384, -0.9575637578964233], [-1.0532536506652832, 0.3748025596141815, 0.09132273495197296, 1.4015612602233887, -0.35543879866600037, -0.17800261080265045, 0.6010520458221436, 0.7237612009048462, 1.4754661321640015, 0.7151020169258118, 1.2637414932250977, 1.0820053815841675, 0.40522679686546326, -1.0588473081588745, -0.3601468503475189, -1.1073498725891113, 3.0973386764526367, -2.0571353435516357, -0.7965030074119568, -0.3916185200214386, 1.4620534181594849, -0.5416353344917297, 0.6013906598091125, 1.9881362915039062, -0.17352436482906342, -0.8532935976982117, 0.7736045718193054, -1.0353929996490479, 0.01584444008767605, -0.6580056548118591, 0.8971956372261047, -0.852066695690155, -0.7071419954299927, 1.621262788772583, 1.9170175790786743, 0.8748470544815063, 0.8856320381164551, -0.29338696599006653, -0.4113021194934845, 0.5065615177154541, -1.2546828985214233, -0.7577869296073914, 1.2564913034439087, 1.4437326192855835, -0.6152552366256714, 0.404057115316391, 0.26488929986953735, 1.4685192108154297, -1.744441270828247, -1.5800186395645142, -1.0515564680099487, -0.21695353090763092, -0.1796075999736786, 0.23790371417999268, -1.592853307723999, -1.4225130081176758, -0.7475113868713379, -0.7569751739501953, 1.354775309562683, -0.3668838441371918, -0.19749803841114044, 1.4610346555709839, -1.8718438148498535, -0.3781191110610962], [0.023067491129040718, -0.6696888208389282, 0.11685693264007568, -1.565600037574768, 1.7433195114135742, -0.8805895447731018, -1.219248652458191, -0.38985559344291687, 0.35345613956451416, 2.156524419784546, 0.53000408411026, -0.21760454773902893, -0.5005965232849121, -0.24338504672050476, 1.857593059539795, 0.5701093077659607, 1.0078709125518799, -1.3065731525421143, -1.211944580078125, -1.8670616149902344, 0.16247330605983734, -0.4292207956314087, -1.6459883451461792, 1.556724190711975, -0.11916690319776535, -0.4817047417163849, 1.696252703666687, 0.6400213241577148, -0.005439102649688721, -0.9626201391220093, -0.07350613176822662, 0.17246712744235992, 0.5169406533241272, 0.14993901550769806, -0.1502138376235962, 1.904986023902893, -0.11605329811573029, 0.2889436185359955, 0.7764402627944946, -2.484696865081787, 0.7053511738777161, 0.7222357392311096, -1.0125828981399536, -0.10617173463106155, 0.14163270592689514, -1.455207109451294, -0.06630496680736542, 2.150275945663452, 1.0027210712432861, -1.0500413179397583, -0.26918551325798035, 0.4020518362522125, -0.7077444791793823, -2.1414010524749756, 1.2598762512207031, 0.7240921854972839, 0.49320778250694275, 0.8595871925354004, 0.21946333348751068, 0.12868081033229828, 0.18490266799926758, -1.6740981340408325, -0.9964994788169861, 0.0477287694811821], [-0.5285111665725708, -0.670356810092926, 1.52459716796875, -0.11096926033496857, 1.1087112426757812, 0.3442353904247284, 0.38264182209968567, 0.74425208568573, -1.3452681303024292, -0.17753735184669495, 1.3060550689697266, 2.0751659870147705, -0.3501904308795929, 1.974553108215332, 0.32138386368751526, -0.1912330687046051, -1.3882514238357544, 1.0482627153396606, 2.2964251041412354, 1.7496964931488037, -0.047406453639268875, 0.0012311170576140285, 0.3847440779209137, -0.5642951726913452, 0.4457547664642334, -0.30866876244544983, -1.2182414531707764, 0.2441299855709076, -1.0498409271240234, -1.0326790809631348, 0.8993954658508301, -0.18641692399978638, 0.5247553586959839, 1.9003612995147705, 0.6834592819213867, -0.2501738965511322, -0.024559585377573967, -0.18965348601341248, -0.48128682374954224, 0.47410938143730164, -1.0583950281143188, -1.9014873504638672, 0.41913360357284546, -0.3366035223007202, 0.9868256449699402, 1.6584131717681885, 0.8620755076408386, -0.49556493759155273, -1.0219647884368896, 0.32945817708969116, -0.30672958493232727, -0.10427957773208618, -1.254671573638916, 0.04776141047477722, 1.0471835136413574, -0.8195812106132507, -0.13108648359775543, 0.6881805062294006, 0.07078652828931808, 1.1915717124938965, -0.5071267485618591, -0.39748120307922363, 1.5702283382415771, -0.6741621494293213], [-1.601905345916748, 0.5367550253868103, 0.7156341075897217, -0.26315954327583313, 0.7257625460624695, 1.241837739944458, 1.6654963493347168, 0.18515798449516296, -0.9237809777259827, -1.842634677886963, 0.8692103028297424, 1.8052915334701538, -0.43640822172164917, 1.191680669784546, 1.654345154762268, -0.40102726221084595, -0.48825007677078247, -1.148209571838379, -2.572967052459717, 1.0541114807128906, 0.929990291595459, -1.5209486484527588, 0.4750821590423584, 0.0033799447119235992, -0.37947478890419006, 0.2382102906703949, -1.9000483751296997, -0.4866326153278351, 0.7996140718460083, -1.5324616432189941, -1.771815538406372, 0.1747702956199646, -0.5025143623352051, 1.5486992597579956, -1.0756127834320068, 0.1270134449005127, -0.12637826800346375, 0.13201306760311127, 0.6822801232337952, -0.6934812664985657, 0.5412794351577759, -1.2320127487182617, 0.1473267376422882, -0.7070935368537903, -0.31801870465278625, 0.8600946068763733, -0.927898645401001, -0.19935595989227295, -2.1660945415496826, 0.6221861839294434, 2.0568313598632812, 3.063793659210205, -0.29954254627227783, 0.28232529759407043, -1.4115245342254639, -0.347374826669693, -1.2640728950500488, 0.44006797671318054, -0.5222780108451843, 0.06798446923494339, 0.19679823517799377, 0.5135117173194885, -0.4285478889942169, -0.011697356589138508], [1.0125311613082886, -1.4654147624969482, -1.0650634765625, -0.47335898876190186, -1.3012428283691406, -0.7838224768638611, -0.11369914561510086, -1.0391087532043457, -0.19034194946289062, 0.054651036858558655, -1.103827953338623, 1.429597020149231, 1.1900146007537842, 1.6035897731781006, 0.05387183278799057, -0.30524566769599915, -0.764572024345398, -0.02320573478937149, 0.43194782733917236, 0.5069172978401184, 1.177346110343933, 1.1314584016799927, -0.3091103434562683, -0.07666152715682983, -0.5136843323707581, -0.9620407819747925, -0.25737065076828003, 0.7724876403808594, -0.9380835890769958, -0.033472031354904175, -1.2792683839797974, -0.6352822184562683, -0.17446504533290863, 0.452263742685318, -0.3272879123687744, 0.6795476675033569, -0.21395717561244965, 1.5894166231155396, 0.032051511108875275, -0.1903846263885498, -0.134428933262825, 2.4112155437469482, -0.12149205803871155, 0.7101572751998901, 0.029947606846690178, 0.04892374202609062, 0.5675845146179199, 1.2568416595458984, 0.5573483109474182, -0.3469315469264984, -0.5439473986625671, -0.34661203622817993, 0.2981400787830353, -0.0469263531267643, -1.8156074285507202, -1.761169672012329, -0.3330903947353363, 1.2738665342330933, -0.6838186979293823, 0.9832810163497925, 0.21751688420772552, 0.1842465102672577, 0.16988252103328705, 1.1794506311416626], [-0.16154073178768158, -0.43551105260849, 0.31158891320228577, 2.185244560241699, 0.21357934176921844, 0.6961977481842041, 0.8053518533706665, -0.27939072251319885, -1.2226121425628662, -0.5986251831054688, 0.549537181854248, 0.8746882081031799, -0.7227321267127991, -0.04303498566150665, -0.6051356196403503, 0.059684738516807556, 1.7926548719406128, 0.2857775092124939, 0.9272305369377136, -0.047518882900476456, 1.779271125793457, -0.5790841579437256, 1.6476397514343262, -0.985416829586029, 1.1545014381408691, -0.7545043230056763, -0.026213370263576508, 0.3752427101135254, 1.6974869966506958, 0.1695336550474167, 1.7610784769058228, 0.7669353485107422, -0.20767827332019806, 1.6110528707504272, 1.7456804513931274, -0.8682672381401062, -1.005252480506897, -1.3677927255630493, -0.6262093186378479, -1.1807310581207275, -1.402768850326538, 1.9918537139892578, -0.6274209022521973, -1.0012061595916748, 0.7076619267463684, 1.8960037231445312, 0.9004409313201904, -2.0972936153411865, -1.2593588829040527, -0.596433699131012, -0.23871004581451416, -0.15082423388957977, 0.39510515332221985, 0.11797194927930832, 0.5106849074363708, -0.6918229460716248, -0.1478865146636963, -0.015829306095838547, 0.730331540107727, -2.2914719581604004, -1.047153353691101, -0.49545833468437195, 0.19433383643627167, 1.4522895812988281], [-0.723526656627655, 0.04666537046432495, 2.3496103286743164, 0.8650177717208862, -0.07386072725057602, -1.4422978162765503, -0.38442984223365784, 0.7022508978843689, 1.488215446472168, -0.5608523488044739, -1.9728895425796509, 1.468725562095642, 1.266343593597412, 0.165452241897583, 0.16441169381141663, 0.20815694332122803, 2.5266621112823486, 0.5380576252937317, -0.06065835431218147, -1.8780208826065063, -0.46770983934402466, 2.0773863792419434, 1.0484693050384521, -0.10523103177547455, -0.02339603193104267, 0.3196173310279846, -0.22385086119174957, 0.8029898405075073, -0.10934042185544968, -0.4806864559650421, -0.14011019468307495, -0.5778951644897461, 0.3125970661640167, -0.5701573491096497, -0.7072659730911255, 1.6750952005386353, -1.8154833316802979, 1.1014324426651, 0.9333516359329224, 0.7415449023246765, -0.23677441477775574, 0.6004103422164917, -0.5786595344543457, -0.8889074325561523, 1.5063220262527466, 0.18457527458667755, 0.35664474964141846, -1.9089486598968506, 0.31448015570640564, 1.2817918062210083, 0.3907070457935333, 0.36506006121635437, -0.25989004969596863, -0.8751841187477112, 0.666335940361023, -3.0859134197235107, -0.5785384774208069, 1.5266640186309814, 2.3258705139160156, -0.26173391938209534, 0.4784260094165802, -0.7062744498252869, -0.7402727007865906, -1.7541660070419312], [1.1836363077163696, -1.3566597700119019, -0.03322817012667656, -0.5267175436019897, -0.0070319389924407005, 0.8430449962615967, -0.8443495631217957, -0.22150161862373352, -0.6021624207496643, 1.72508704662323, 0.029792113229632378, 0.12739700078964233, -0.12614285945892334, -0.2940647602081299, 0.5021975040435791, 0.2741963863372803, 0.5063245892524719, -2.2998838424682617, 0.9761869311332703, -0.44584816694259644, 0.7912811040878296, -0.9664296507835388, 0.7986757755279541, 0.942055344581604, 1.293411135673523, -1.3693342208862305, -0.8927714824676514, -0.4660804271697998, -0.037570513784885406, -1.6534919738769531, -2.070983409881592, -0.09453171491622925, -0.48707059025764465, -1.4835526943206787, -0.13106945157051086, 1.0085660219192505, -1.5929598808288574, -1.1206040382385254, 0.6695137023925781, -1.188321828842163, 1.186087965965271, -1.484559178352356, -0.46703284978866577, 1.233012080192566, 0.5434793829917908, -0.2798299193382263, 1.7005537748336792, 1.3396787643432617, -0.028431668877601624, -0.01862150989472866, 0.8849961757659912, -0.9346381425857544, -2.3791213035583496, 1.7029842138290405, -0.027520986273884773, 1.517244815826416, -0.5012678503990173, 0.4407621920108795, -1.4189256429672241, 0.602297842502594, -1.5780963897705078, -1.2416030168533325, 2.0908896923065186, 0.7154814600944519], [-0.8042712807655334, 0.21032430231571198, -2.831268072128296, 0.576271653175354, 2.4422411918640137, -1.2941011190414429, 0.6078258752822876, 0.5791561603546143, 0.6364961862564087, 0.18732242286205292, 1.076137900352478, 0.29083317518234253, 1.0062711238861084, -0.1789523810148239, 1.7063673734664917, -0.9143725633621216, -0.28641411662101746, 0.017223365604877472, 0.9283250570297241, -0.6844281554222107, 0.1762024313211441, -0.4295010268688202, -0.55040442943573, -0.4212630093097687, 0.5688220262527466, 1.5344023704528809, -0.4596719741821289, 0.1793675571680069, 0.328103631734848, -1.351224660873413, -1.309735655784607, -1.0289934873580933, 1.118940830230713, 1.2235572338104248, 1.4840927124023438, 0.22792963683605194, 0.7744339108467102, -0.5661202669143677, 1.727813482284546, -2.8751306533813477, -0.28162702918052673, 1.3650850057601929, 0.0797819197177887, -1.7660868167877197, 0.4270854890346527, -0.16685903072357178, -0.5495012402534485, 1.1150680780410767, -1.128002405166626, 0.45270484685897827, -0.5761010050773621, -0.9864580631256104, -0.41507285833358765, 0.4749874472618103, 0.022218862548470497, 0.5285693407058716, 0.8471375703811646, -1.3981306552886963, -1.8528332710266113, 1.8672646284103394, 0.5316147804260254, -0.6434714198112488, -1.6435484886169434, 0.19189755618572235], [-0.7224650382995605, -0.028421366587281227, 1.4342265129089355, 0.986814558506012, 1.4640445709228516, 1.7453436851501465, -0.12193717062473297, 0.7173835039138794, -0.9135417938232422, -0.3585968017578125, 1.3916572332382202, 0.24148277938365936, -0.125487819314003, -0.72510826587677, -0.17767642438411713, -0.07797861099243164, 1.5357632637023926, -1.5019234418869019, -0.9931719303131104, -0.7193257212638855, -0.10287319868803024, 0.013713953085243702, -0.43427973985671997, 0.7299931645393372, 0.4696665406227112, 0.6230011582374573, -1.2375917434692383, -2.046576499938965, 1.834054946899414, -2.550119161605835, 0.3800507187843323, 1.2533284425735474, 1.2802338600158691, 1.5619409084320068, 0.16648730635643005, 0.7659918665885925, -1.8884599208831787, -0.9951470494270325, 0.839985728263855, 1.2132902145385742, -0.06795556098222733, 1.5889376401901245, -1.673643946647644, 1.0710599422454834, -0.43938928842544556, -0.42870306968688965, 0.12587782740592957, 0.6635417342185974, -0.4748469293117523, -1.3255687952041626, -0.594327449798584, -0.5694117546081543, 0.36034390330314636, -0.38317766785621643, -0.12388517707586288, 0.30091995000839233, -0.11550324410200119, -0.7540278434753418, -1.0305663347244263, 0.3177560269832611, -0.35506072640419006, 0.11762949079275131, -2.1921467781066895, -0.5256414413452148], [-0.5324328541755676, 0.8834527730941772, 0.5010803937911987, 0.08606280386447906, -1.7307145595550537, -0.3658103346824646, 0.8045778274536133, 0.576781153678894, 0.04984413459897041, 0.7220525741577148, 1.6319948434829712, 1.58452308177948, -0.11270052194595337, 0.3025408983230591, 0.21356160938739777, -0.6888872981071472, -1.0199486017227173, 0.8074355125427246, 1.7384440898895264, -0.5230261087417603, 2.034477472305298, -0.5258785486221313, -1.7885233163833618, -1.259109377861023, -0.2657313346862793, 3.4452006816864014, 0.4784056842327118, 0.39471378922462463, -0.9995518922805786, -1.2557364702224731, 2.312732458114624, 0.3269420862197876, -1.2407538890838623, -0.1373916119337082, -0.5576236844062805, -0.06291648000478745, 0.11334776878356934, -0.5414953827857971, -0.20362579822540283, -0.13725034892559052, 0.4581972062587738, -0.6765257716178894, -0.7324780225753784, -0.06092449277639389, -1.6511263847351074, 0.7060173749923706, -0.24943377077579498, 0.8806782960891724, 2.1253881454467773, 1.9133505821228027, 0.39952972531318665, -1.2485769987106323, -0.1909661740064621, 0.1510283648967743, -0.20570071041584015, -0.6542690396308899, 1.659132957458496, -1.1938774585723877, 0.8219132423400879, 0.2296629101037979, -0.41611215472221375, -0.03392671048641205, 0.9638781547546387, 0.2531678378582001], [-0.7151461839675903, 1.7240580320358276, -1.7634241580963135, -1.6713205575942993, -0.35233891010284424, -1.260547161102295, -0.2773456573486328, -2.022942066192627, -0.23726457357406616, 0.6885994076728821, -0.009119966067373753, 1.8666287660598755, -2.29070782661438, -1.905413269996643, 0.06281746178865433, 0.31981611251831055, 0.02042127586901188, -0.8225055932998657, -0.6915650963783264, 0.043292224407196045, 1.369053840637207, 1.058689832687378, -0.5740953683853149, -1.3823503255844116, 0.836391806602478, -0.6248847246170044, 0.7216088175773621, 0.409719318151474, -0.2542661130428314, 0.7836610674858093, 1.1246395111083984, 0.20588910579681396, -0.47396320104599, -0.48403945565223694, -0.8807235956192017, 1.2796225547790527, -0.34960153698921204, 0.45152589678764343, -0.7909551858901978, 0.7459276914596558, -0.4092963635921478, -0.020491497591137886, 0.555029571056366, 0.8803032636642456, 0.8226313591003418, 0.7756571769714355, 1.3629764318466187, -0.6373758316040039, -0.05314110219478607, -2.606365919113159, -0.019582664594054222, 1.3834556341171265, 0.6799511909484863, -0.7233428359031677, 1.2080668210983276, -2.107757329940796, -0.0980408638715744, 0.8093545436859131, -0.2421547770500183, -0.795724630355835, 0.7986345291137695, -0.46266576647758484, 1.2760114669799805, 0.15667282044887543], [-0.5126522183418274, -0.012030943296849728, 0.26083800196647644, 0.7870197296142578, 0.8131401538848877, -1.1474616527557373, 0.5893909931182861, -1.0592845678329468, 0.016351968050003052, 0.13389110565185547, 0.2920377552509308, 2.415123462677002, -0.2078711986541748, -0.1032315120100975, -0.03206564486026764, 0.6762704253196716, 0.7849056124687195, -0.1997908651828766, -2.599513292312622, 0.09035668522119522, 0.6578171253204346, 1.230200171470642, -0.5628007650375366, 0.063508041203022, -0.9495444297790527, 1.2889107465744019, -2.354220390319824, 0.5343017578125, -0.07773026078939438, 0.29333484172821045, 0.7517622709274292, -0.749392569065094, 0.17414472997188568, -1.6667546033859253, -0.5805664658546448, -0.20557966828346252, -0.8757873177528381, -1.6933733224868774, -1.9885175228118896, -0.2070642113685608, -0.3148622214794159, 0.2697598338127136, 1.6305001974105835, -0.36241739988327026, 1.3865281343460083, -0.1406589299440384, 0.4979429543018341, 0.29313674569129944, -1.5234373807907104, -0.7563825845718384, 0.4977743327617645, 1.6706006526947021, -0.6408596038818359, 0.7257025837898254, -2.1307811737060547, -1.5891919136047363, 0.6974571943283081, 0.7613906860351562, -0.10467389971017838, -0.5527438521385193, 0.40521568059921265, -1.2748013734817505, 1.3807134628295898, 0.8250114917755127], [-0.19424375891685486, -1.7229140996932983, -0.919085681438446, -0.9578062295913696, -0.47000208497047424, 1.6534850597381592, 0.5228310823440552, -1.1287473440170288, -2.146225929260254, 0.19308717548847198, -0.3073217570781708, 1.4382996559143066, -2.2973976135253906, -0.2009354531764984, 0.17708095908164978, -0.4097384214401245, 0.9808821678161621, -1.5277572870254517, 1.977327585220337, -1.1855592727661133, -0.3842988908290863, -0.9447697401046753, -1.9052749872207642, -1.5916203260421753, -1.7394126653671265, -0.38463419675827026, 0.8181148171424866, 1.7451151609420776, 0.5763932466506958, 0.1193358525633812, 0.5147197246551514, 0.2001393735408783, -0.15523236989974976, -0.4686339199542999, 1.4944264888763428, -1.1352430582046509, 0.7847464084625244, 1.2098819017410278, -0.055275000631809235, 1.026291012763977, 0.032931335270404816, 0.14596211910247803, -1.0069185495376587, -0.7714483141899109, -0.9383295178413391, 1.0623104572296143, 1.3803542852401733, -1.2384467124938965, -1.3344255685806274, 0.16825290024280548, -1.1405996084213257, 0.3780657947063446, -0.34630441665649414, -0.9739691615104675, 2.1745223999023438, 0.8024322390556335, 0.4167334735393524, 0.04063377529382706, 0.18231023848056793, 0.3764335811138153, 1.7827210426330566, -1.7816731929779053, 1.1683251857757568, -0.7673701047897339], [-0.8627845644950867, -0.04622993990778923, -0.10688623040914536, -0.03320802375674248, -0.7098070383071899, 0.433713436126709, 1.066478967666626, -0.6100709438323975, -0.9120597243309021, -0.9868427515029907, -0.30582302808761597, 0.08285452425479889, 0.12679430842399597, 0.4465535283088684, 0.1476462334394455, 0.9371048808097839, 0.8295363187789917, 0.6782484650611877, 0.36646929383277893, 0.5130437016487122, -0.9695615768432617, -0.358550488948822, 0.9146338105201721, 0.795529842376709, -0.3902173936367035, -1.4170037508010864, -1.565050482749939, -0.3563125729560852, 0.89092618227005, 0.6950355172157288, -1.141934871673584, -0.25321707129478455, -0.12025803327560425, -0.2086125910282135, -0.19370470941066742, 0.07319392263889313, -1.1908084154129028, -1.5095971822738647, 0.07759172469377518, -0.2025689333677292, -0.1325533539056778, 0.3374004065990448, 1.2101994752883911, -1.1023120880126953, -0.4844914674758911, -1.4078564643859863, -0.6217362880706787, -0.13812221586704254, 0.8892413973808289, -0.23414814472198486, 0.024363376200199127, 1.4961457252502441, -0.2597733438014984, 0.1072002574801445, -0.11842568963766098, -1.8892006874084473, 0.4576842486858368, 0.11795428395271301, -0.20651040971279144, 0.3124552071094513, 0.7167486548423767, -1.0920085906982422, 0.99271559715271, -1.1932733058929443], [-1.2193149328231812, 0.19987432658672333, -0.4541723430156708, -1.1880004405975342, -0.1476520448923111, -1.2096912860870361, 1.062079668045044, 0.6618170142173767, -0.15617744624614716, 0.3047970235347748, -0.6709017753601074, 0.947489857673645, -0.8478712439537048, 1.2620593309402466, -0.0318874791264534, 0.2545447051525116, -0.253730833530426, -0.9804956316947937, -0.7012327313423157, 0.8985838294029236, 1.6085656881332397, 0.08187486231327057, -0.3446296751499176, -0.2831382155418396, 1.400348424911499, -0.08902163803577423, -0.06802776455879211, 0.15277500450611115, -0.5157387256622314, 0.146673783659935, -1.639229416847229, 1.4209914207458496, -0.8376564383506775, -0.392343133687973, 0.6147955656051636, -0.045643117278814316, 0.5074028968811035, -0.3201555609703064, -0.657665491104126, -1.6518850326538086, -0.7697844505310059, 0.8343576788902283, -0.07573876529932022, -0.3130234479904175, 0.8117328882217407, -1.0948866605758667, 0.044310569763183594, -0.4557884633541107, -0.9945143461227417, -0.48074644804000854, 1.591248631477356, 1.4579188823699951, 0.5237652063369751, -0.040570080280303955, -1.3517667055130005, 1.0061469078063965, -0.7808661460876465, 1.9048782587051392, -0.05117618665099144, -1.3409793376922607, -1.6918915510177612, -1.0129576921463013, 0.1640467643737793, 1.3632436990737915], [1.0285329818725586, -0.3594626486301422, 1.1962403059005737, -1.0822659730911255, -0.8366209864616394, 1.9266895055770874, 1.5048800706863403, 0.014973112381994724, -0.31765860319137573, 1.3956669569015503, -0.24799953401088715, 0.10850881040096283, 0.6612109541893005, -0.4375287592411041, -0.12118571251630783, -1.0751537084579468, 1.2113666534423828, -1.426623821258545, 1.8579370975494385, -0.7419112324714661, 0.38646838068962097, -0.528503954410553, -0.18937014043331146, 0.25164103507995605, -0.2627831995487213, -0.34734269976615906, 0.7808504104614258, 1.4148308038711548, -0.2215997725725174, -0.7097359895706177, -1.281346082687378, -0.4817981421947479, -1.0612671375274658, 1.4646638631820679, -0.3567301034927368, 0.3094936013221741, 1.296866536140442, 0.1972704529762268, 0.5820201635360718, -0.5705583095550537, -0.5518261790275574, -1.0913227796554565, 1.512685775756836, -1.3443347215652466, 0.6303719878196716, 1.2641184329986572, -1.0384531021118164, -0.3580198585987091, 0.40868079662323, 0.05739368870854378, 0.6656193733215332, -1.8170613050460815, 0.5966704487800598, 0.6595373749732971, -0.5538443326950073, 0.04574766382575035, -1.237149953842163, -0.6297626495361328, -0.3585885465145111, -0.6780462265014648, -0.2816942632198334, 0.13534969091415405, 1.1984189748764038, 1.6395729780197144], [0.8568140268325806, 0.13269992172718048, -0.6347165703773499, 0.5578966736793518, 0.41703951358795166, -0.5180988311767578, -0.6032899618148804, 1.6760594844818115, -0.3405205309391022, 0.10395688563585281, -0.18771511316299438, 0.30140548944473267, -0.4005398452281952, 1.0564697980880737, -0.6633794903755188, 1.0814330577850342, -1.6022177934646606, 0.40301042795181274, -0.5463431477546692, -1.5584206581115723, -0.20723885297775269, -1.042190670967102, -0.22254613041877747, -0.5397850871086121, -1.8841218948364258, 1.0614835023880005, 0.038698479533195496, -0.24858492612838745, 0.1563337743282318, -1.451672911643982, 0.37543734908103943, -0.8670128583908081, 0.1482122838497162, 0.6130406856536865, -1.2674814462661743, 0.03472619876265526, 1.3733664751052856, 1.6554094552993774, 0.07187347859144211, 0.30020537972450256, -0.5984072685241699, -0.6550381779670715, -1.5151057243347168, -1.1450048685073853, -0.014336778782308102, -0.7274487018585205, 0.9430350661277771, 0.06223273277282715, 0.6203998327255249, -0.673632800579071, 1.2158360481262207, -0.5429289937019348, 1.0512264966964722, -1.1363334655761719, -1.3268386125564575, -1.623781442642212, -1.2954604625701904, 1.2611708641052246, 0.3165266513824463, 1.9443953037261963, -1.1413450241088867, 0.816587507724762, -0.5230422019958496, -1.071180820465088], [1.7249150276184082, -0.19789287447929382, -0.20200005173683167, 0.5732021927833557, -0.020089933648705482, -0.9539791345596313, -1.105308175086975, -0.3486926853656769, -0.58050537109375, 1.2320940494537354, -0.5000735521316528, -0.7801487445831299, 0.11452703922986984, -0.866357684135437, 0.31994888186454773, -1.0625756978988647, 0.5774729251861572, 0.419520765542984, -0.2539334297180176, 1.6938247680664062, -1.2176456451416016, 1.2444754838943481, 0.8314586281776428, 0.7234395742416382, 2.319859504699707, 1.1696414947509766, -0.034532953053712845, 0.7391358017921448, -0.9755626320838928, -0.3978964388370514, -2.0280508995056152, 0.1435931771993637, 1.2293789386749268, 0.1605762541294098, -0.37929603457450867, 0.9385096430778503, -0.4099361300468445, -0.4542893171310425, -2.1490464210510254, -2.3370418548583984, -0.3929900527000427, 0.382414847612381, 0.4537543058395386, -0.9885097742080688, 0.011100203730165958, -1.7701523303985596, -0.2137039601802826, -0.4194258749485016, 0.05369365215301514, 0.2671699523925781, -0.13337299227714539, 0.05777834728360176, 0.4319978654384613, 1.7498067617416382, -1.3141765594482422, 1.1373071670532227, 1.1140044927597046, -1.4547911882400513, 0.5983344316482544, -1.205626130104065, 0.26596277952194214, -1.3917064666748047, 0.4024747908115387, -0.17714367806911469], [1.1848996877670288, 0.7436050176620483, -0.583446741104126, 1.1396054029464722, 2.397611618041992, -2.2144150733947754, -1.7329906225204468, -0.9546676874160767, 0.6678019165992737, -0.5044218897819519, -1.015012264251709, 0.557083249092102, 0.7905747890472412, 0.3025813698768616, 0.3611154854297638, -1.5583399534225464, -0.5474418997764587, -0.8633479475975037, -0.09723781049251556, 1.0538896322250366, -0.7421074509620667, 0.6571280360221863, -1.8139281272888184, -0.44616663455963135, -1.4751787185668945, -0.561657726764679, -1.2804539203643799, 0.09606306999921799, -0.8353196978569031, -1.4649710655212402, 0.020785527303814888, 0.715722918510437, -1.2651509046554565, 0.32654428482055664, 0.8153334259986877, -0.8957595825195312, -1.3387380838394165, -1.5568037033081055, 0.30681461095809937, 1.0817525386810303, 1.167726993560791, -0.9898217916488647, 0.044653818011283875, -1.3787362575531006, -1.0223687887191772, -0.19932584464550018, -1.5071266889572144, -0.6944875121116638, 1.0252821445465088, 0.41433846950531006, -0.5810925364494324, -0.18133626878261566, 0.07146260142326355, 2.172060489654541, 0.6651642322540283, 0.8941068053245544, -0.039294660091400146, 0.3142469823360443, -0.9625489115715027, -0.011134105734527111, 0.3794887363910675, 0.45699572563171387, 0.9259185791015625, -0.36659765243530273], [0.5286223888397217, -0.32006317377090454, 0.9185985922813416, 0.5229753255844116, 0.16903682053089142, 0.03212525695562363, -1.7364006042480469, -0.16361460089683533, -0.7904173135757446, -0.3394047021865845, 2.2892889976501465, -0.6676420569419861, -0.222897008061409, -1.704902172088623, -0.44128865003585815, -0.6608902812004089, -0.8506059646606445, 0.6424286365509033, 0.1877465546131134, 0.14126408100128174, -1.5365426540374756, -0.3474741280078888, -1.531406283378601, 0.8004525899887085, 0.650264322757721, 0.4260304272174835, 0.7923719882965088, 0.48838794231414795, -1.155052900314331, 1.5133224725723267, -0.11280597746372223, 0.91232830286026, 0.12131915986537933, -1.2302937507629395, 0.3951861262321472, 1.0241756439208984, 1.8118051290512085, 0.2715449929237366, -0.8424281477928162, 0.47494229674339294, 1.440616488456726, 0.10833603143692017, 0.7565991282463074, 0.07848899066448212, 1.8176268339157104, 1.131822943687439, -1.8144482374191284, 0.24145832657814026, -0.6587374806404114, 0.9351481795310974, 0.88701331615448, 1.4853270053863525, -1.8661011457443237, -0.9265180826187134, -0.12206480652093887, -1.289510726928711, -0.37667691707611084, -2.7181971073150635, -1.1177209615707397, 1.1324212551116943, -0.06625238806009293, 1.6806097030639648, -0.4990319013595581, 0.2787676751613617], [-2.743436336517334, 2.1177680492401123, 0.19335393607616425, 0.6397911310195923, -1.5377675294876099, -1.614119052886963, 1.0850659608840942, 0.39510688185691833, -0.24078206717967987, 0.5197151899337769, -1.171981930732727, -0.164182648062706, -1.3593021631240845, 1.4832162857055664, 1.1929835081100464, -0.10898289829492569, 0.18792663514614105, 2.050997257232666, 0.636018693447113, 0.9702009558677673, 0.27480292320251465, 0.8293603658676147, -2.0990138053894043, -1.1915404796600342, -0.08912370353937149, -1.2068177461624146, -0.6556132435798645, -0.9134820699691772, -1.763731598854065, 1.2367134094238281, -0.77433180809021, -0.8974181413650513, -1.118838906288147, 0.17327575385570526, -0.7070178985595703, -0.49898308515548706, 0.0933600664138794, -0.3461737334728241, 0.2685258090496063, 1.560578465461731, 0.47655895352363586, -0.1405564695596695, -0.42389675974845886, -0.314233660697937, -0.61167973279953, 0.2089426964521408, -0.48304733633995056, 0.9142877459526062, 0.5856595039367676, -0.32615500688552856, -0.15736903250217438, 0.6375671029090881, 0.6228392124176025, 0.5674319863319397, 0.43956148624420166, -0.6798335909843445, 1.5516659021377563, 0.4406152069568634, -0.2448301613330841, -0.1459996998310089, 0.4159044325351715, 2.3364040851593018, 1.7812455892562866, 0.18399614095687866], [-0.005675585009157658, 1.151453971862793, -0.09905514121055603, 0.12206009775400162, 1.5686498880386353, 0.8082814812660217, 1.238389492034912, 0.6094850301742554, 1.0603305101394653, 0.2689276933670044, 0.019881078973412514, 2.317006826400757, 0.4394415020942688, 1.4926955699920654, -0.1581825315952301, 0.15091463923454285, 2.313363552093506, 2.2761595249176025, 0.9966892600059509, 0.07233031839132309, -0.3810487687587738, -0.40166935324668884, -0.9110425114631653, 0.4416063725948334, 1.0434224605560303, 0.965446412563324, -0.03650812804698944, -0.8837196230888367, 2.88309907913208, 0.3952203094959259, 2.4723336696624756, -0.456390380859375, 1.863983154296875, -0.3201926350593567, 0.6219831705093384, -0.34542325139045715, 0.7993019223213196, -0.849774956703186, 0.5474711656570435, -0.3171948790550232, -0.7752358913421631, -0.5079677700996399, 1.0104988813400269, 0.4381265640258789, 0.1705075055360794, -1.2465986013412476, -1.6561304330825806, -0.4170018434524536, -0.7064104676246643, -0.2937549948692322, -0.6129836440086365, -0.09246262907981873, 0.4671807587146759, -1.138518214225769, -1.14399254322052, -0.1312415450811386, 0.27221861481666565, -0.4137513041496277, 0.06170431151986122, 1.4011712074279785, -0.91061931848526, 0.9599026441574097, 0.7709612250328064, -1.001904010772705], [1.2531119585037231, -0.5445745587348938, 0.06124546378850937, 1.6288262605667114, 0.05949729308485985, 0.9631369113922119, -1.3386378288269043, 0.21428215503692627, 0.5858274102210999, 0.5064777135848999, -1.3361419439315796, -0.32016491889953613, -0.2513861358165741, -0.8569115996360779, -0.4905211329460144, 0.2139245867729187, 0.005792378447949886, -0.29167279601097107, -0.1463463008403778, -2.354933023452759, -0.8675234913825989, -0.7042043805122375, -0.3531425893306732, 0.39676666259765625, 0.6382782459259033, -0.4186520278453827, -0.4356403052806854, 0.6063993573188782, -0.7909881472587585, -0.6814195513725281, 1.1744611263275146, -1.2228065729141235, -0.2864236533641815, 1.1862635612487793, 0.4562779366970062, -0.9165841937065125, 0.5293409824371338, -0.7593091726303101, 1.556776762008667, -0.24259188771247864, 0.24385254085063934, 0.11916911602020264, -0.6314646601676941, 0.12738198041915894, 2.0466389656066895, -0.08196282386779785, -0.8698740601539612, -1.1171163320541382, -0.12465536594390869, 1.598476767539978, 0.6980048418045044, 0.8324485421180725, 0.68119877576828, 1.2465792894363403, -0.6123139262199402, -0.45743709802627563, 1.7779054641723633, -1.556396484375, -1.5277764797210693, -0.37030893564224243, -1.3710079193115234, -0.4184543788433075, 2.3536016941070557, 1.8090373277664185], [0.9216878414154053, -0.04613301530480385, -0.6511552929878235, -1.0668522119522095, 0.3917411267757416, -0.6855354309082031, 0.24554543197155, -1.17388117313385, 0.29943323135375977, 0.21977734565734863, -1.910407543182373, 0.4955637753009796, 1.1700724363327026, -0.7961152791976929, -0.24408528208732605, 0.043450821191072464, -1.3557790517807007, -0.6036608219146729, -0.5605877041816711, 0.21333442628383636, -1.2340126037597656, 0.9346739053726196, -0.18093353509902954, -0.7402702569961548, 0.7963898777961731, 0.18762722611427307, 1.0677014589309692, 0.5087505578994751, -0.39690691232681274, -0.7971290349960327, -0.240391343832016, 0.27145206928253174, 0.2402617186307907, 0.16829216480255127, 0.1653042435646057, 0.7284718155860901, 2.0958003997802734, -0.3005232512950897, 0.31969571113586426, 0.5225971341133118, -0.3357076048851013, -0.8694194555282593, -0.5030351281166077, 0.2586669921875, 0.5710688233375549, 2.878788709640503, 1.0702576637268066, -1.7473230361938477, 0.8325669765472412, -0.49602994322776794, 0.4302477240562439, 1.129014492034912, -0.5908659100532532, -0.3520243465900421, 0.43655893206596375, -0.05108672380447388, -0.6065283417701721, -0.7275476455688477, 1.530969262123108, -1.3834631443023682, 0.963566243648529, -0.674429714679718, -0.3157140910625458, -0.4450569152832031], [-2.19047212600708, -0.5592501759529114, -0.8645538687705994, 0.036834705621004105, 1.0295003652572632, 1.7110233306884766, 1.3282307386398315, 0.06200188025832176, -0.7240232229232788, 0.3042740821838379, -0.8613578081130981, 0.26077625155448914, -1.8744436502456665, -0.7226643562316895, -0.21290293335914612, 0.23381869494915009, 0.43480610847473145, -0.9134165644645691, -0.7845058441162109, 0.6683973073959351, 2.0074210166931152, 0.7591086626052856, -0.9654722213745117, 2.154705047607422, 1.953413486480713, 0.5747796893119812, 0.5532081723213196, -1.4125638008117676, -0.06939613819122314, 0.513766348361969, 0.14263051748275757, -0.7666172981262207, -0.2817773222923279, -0.706741988658905, -0.04018184170126915, 0.7501853704452515, 0.4414302110671997, 0.3947972357273102, 0.316707044839859, 0.5305414199829102, -0.0069686248898506165, -1.0835551023483276, 0.2912871241569519, -0.5380598306655884, 0.5342184901237488, -0.30346712470054626, 0.005853474140167236, -1.1862146854400635, -0.7567132711410522, 0.49746713042259216, -0.18425960838794708, -1.345252275466919, 0.05045479163527489, -0.4364103674888611, -0.7198262810707092, 0.7941671013832092, 0.08244805783033371, -2.008601188659668, -0.6250490546226501, -0.7520378828048706, -1.4861756563186646, 0.08049748092889786, 0.2186129093170166, 0.24839916825294495], [0.7504053115844727, -2.491725206375122, -0.06364884972572327, -1.6462270021438599, 0.6337481737136841, 0.9382064342498779, -0.05353948846459389, 0.5375684499740601, -0.26257696747779846, 0.5491176843643188, -0.2348785698413849, -0.816307544708252, -1.2548919916152954, 0.0032755674328655005, 0.01723412796854973, -1.0092828273773193, -1.0619556903839111, -0.4854114353656769, -0.4982161819934845, 1.4899286031723022, -1.3953098058700562, -0.5880282521247864, 2.0399041175842285, -0.45820844173431396, 0.5624080896377563, -0.833214521408081, 0.6659383177757263, -0.16726934909820557, -0.9681074619293213, 0.726402997970581, 1.983817219734192, 1.506986379623413, 0.7778943181037903, 0.7384657263755798, -0.3496512472629547, 0.9336879849433899, 0.6517682075500488, -0.7759889364242554, -1.3968687057495117, -0.33463186025619507, -0.1339380443096161, 0.6510070562362671, -1.11509370803833, -0.5821457505226135, -1.367139220237732, -1.4220209121704102, -0.5556187033653259, 0.2756398916244507, 0.7768663167953491, 0.2991552948951721, 0.6718159317970276, 0.45699289441108704, -0.19967222213745117, -0.5985305309295654, -0.2505386769771576, 0.8508983254432678, 1.9324638843536377, 0.3833213746547699, 0.2792346179485321, -1.0998811721801758, -0.034879960119724274, -0.34396156668663025, -0.24055536091327667, -0.5392982959747314], [1.866348147392273, 0.6129425168037415, 0.6429843902587891, -2.029829978942871, 0.39786916971206665, 0.5976653099060059, -0.6250645518302917, -0.9495936036109924, 0.5827773809432983, -0.7166664600372314, 0.08308250457048416, 1.589046835899353, -0.2396104335784912, -0.7055738568305969, 1.509186863899231, 1.0551859140396118, -0.3781680762767792, 1.1034002304077148, 1.157211184501648, 0.4662169814109802, -0.39980772137641907, 0.3709620237350464, 0.0557895302772522, -0.6394696235656738, -0.47698473930358887, 0.6638516187667847, -2.244643449783325, 0.12090875208377838, -0.29728201031684875, 0.12719237804412842, 0.29328107833862305, 0.1417265087366104, -0.3741292655467987, -0.14300526678562164, 0.7032000422477722, 2.6833109855651855, 1.2802789211273193, 2.2897789478302, 0.13098815083503723, 0.40503618121147156, -0.7059041857719421, 0.9635969996452332, -2.0019538402557373, -0.01632751151919365, 0.5644248723983765, 1.0948842763900757, 0.3208673894405365, 1.133404016494751, -0.07427044957876205, -0.6982781291007996, -0.15069305896759033, -0.7715325355529785, 0.5422489047050476, 1.0031489133834839, 0.2936910092830658, -0.8648332357406616, 0.39345765113830566, -0.11376825720071793, -2.490854024887085, 0.6403111815452576, 0.4363797605037689, 0.9872981905937195, -1.4435861110687256, 1.2524172067642212], [1.3689727783203125, -0.08985643088817596, -1.2945952415466309, 0.1445603221654892, 0.05149413272738457, -1.0154900550842285, 1.4712051153182983, -1.042744755744934, -1.452828049659729, 1.2883809804916382, 2.231428623199463, 0.21830615401268005, 0.4128958582878113, 0.2695300877094269, -0.2693631649017334, 1.9382140636444092, -0.6363263726234436, 0.761829137802124, -0.7241278886795044, 2.28470778465271, -2.8051037788391113, 0.11761021614074707, -0.11650346964597702, 0.5314139723777771, -0.48330169916152954, -0.21576379239559174, -0.4765629172325134, -0.13663172721862793, -0.9138042330741882, -0.46024057269096375, 0.7656079530715942, -0.1926889568567276, -0.5301668047904968, 0.24033311009407043, 1.9097579717636108, -1.4337307214736938, 0.27490416169166565, -0.4501686990261078, -0.00975003745406866, -1.0797700881958008, -2.012214422225952, -0.47864606976509094, 1.4930707216262817, -0.5451714396476746, -1.5617514848709106, 0.15883243083953857, 0.3289870619773865, -0.02105635404586792, 0.10089493542909622, 0.7262228727340698, 2.101316452026367, -0.19284988939762115, -1.4978883266448975, -0.4206472635269165, -2.331484079360962, 1.2187433242797852, 0.6532921195030212, -1.782163381576538, 1.6382935047149658, 0.8136611580848694, 0.08278107643127441, 0.12153227627277374, 0.32698798179626465, -0.8029233813285828], [0.47567883133888245, -1.6328543424606323, 0.6127784252166748, 0.03366759419441223, -0.700201690196991, -1.002125859260559, -1.0776008367538452, 0.30928969383239746, 0.062354277819395065, 1.2876321077346802, -1.949998140335083, 2.7122509479522705, 0.8755238056182861, 1.8486117124557495, 0.8414559960365295, -0.39023643732070923, -1.148848056793213, -0.4036707878112793, 0.7011487483978271, 0.18694691359996796, -1.467082142829895, 0.12623728811740875, 1.3557000160217285, -0.033169277012348175, -1.7930701971054077, 0.24900206923484802, 0.1506158858537674, -1.1437968015670776, 1.0313053131103516, -0.6454092860221863, 0.5685625672340393, -0.12683923542499542, -2.160611391067505, -0.538113534450531, 0.16029143333435059, -1.233034372329712, 1.1400611400604248, -0.716934859752655, 0.3124571442604065, 0.12054169923067093, -0.7357566952705383, 0.48768678307533264, 0.2975323498249054, -0.2342396378517151, -0.8908090591430664, 1.5903699398040771, -0.8072367310523987, 0.42382657527923584, -0.8197526931762695, 0.6324536800384521, 0.18559426069259644, -0.44919922947883606, 0.6692410707473755, -1.5252302885055542, 0.5669093132019043, 1.6096417903900146, -0.2920335531234741, 1.4023538827896118, -0.4037150740623474, 0.7815720438957214, 1.070818543434143, -1.3965781927108765, -0.3766933083534241, 0.48038449883461], [0.8113951683044434, 0.9022378325462341, 0.07168788462877274, 0.6897781491279602, 1.472988486289978, 1.9215021133422852, 1.324725866317749, 1.4277461767196655, 1.9352067708969116, -0.5697557330131531, 1.5684844255447388, -1.3218203783035278, 0.7170262336730957, 1.07315194606781, 0.30707526206970215, 0.18508267402648926, -0.10221506655216217, 0.6157369017601013, 1.9184337854385376, -0.7572712302207947, -0.24492569267749786, 0.20964665710926056, 0.5715808272361755, 0.6788968443870544, 1.424193024635315, -0.58017897605896, 0.8193225264549255, -1.582466721534729, -1.5433787107467651, 0.8131116032600403, 1.0845868587493896, 0.3107438385486603, 0.6359516382217407, -1.1016720533370972, -0.20155607163906097, -0.7686409950256348, 1.2858959436416626, -1.4391462802886963, 0.4177519381046295, 0.03323609009385109, 0.26478466391563416, -1.1019392013549805, -0.5274789929389954, -0.44932907819747925, 0.2720065414905548, -0.056111712008714676, 1.4244470596313477, 0.11835927516222, 1.9213217496871948, 0.2841891050338745, -0.724901556968689, 0.43362200260162354, 1.1859779357910156, 0.2445666640996933, -0.8879076242446899, 0.6968734264373779, -0.16561979055404663, 0.3093894422054291, 0.35903850197792053, -1.1079285144805908, -0.22783039510250092, -0.8664901256561279, -1.8105950355529785, -1.200352430343628], [0.35681191086769104, 0.3148457407951355, 0.7289209961891174, -0.21689461171627045, -0.6899959444999695, -0.35706278681755066, -1.4135065078735352, -0.0274859257042408, 0.16726095974445343, -0.535727858543396, -0.09673932194709778, -0.8577258586883545, -1.281177043914795, 0.28234902024269104, 0.49863237142562866, -2.0235190391540527, -0.3959645628929138, 0.6418846249580383, -0.03722633793950081, -0.24582386016845703, -0.21381817758083344, -0.9987983703613281, -0.8683812618255615, -0.14674103260040283, -1.081405520439148, -0.37711262702941895, -1.1198599338531494, -0.18107903003692627, 0.16845139861106873, -0.7377045750617981, 0.21527329087257385, -0.9595484733581543, -0.428824245929718, -0.4076496362686157, -1.337037205696106, -0.7818324565887451, -0.2614443898200989, -0.4517941474914551, 1.497154951095581, -0.22211134433746338, 0.46824008226394653, -2.1839733123779297, -0.8885315656661987, -0.8677355647087097, 0.20666608214378357, -0.8089274764060974, -0.5362017750740051, 2.4957194328308105, -1.6527360677719116, -1.8049321174621582, 0.7490526437759399, -0.32468488812446594, 0.28230950236320496, -1.0564910173416138, 1.0457355976104736, -0.35346296429634094, 1.1189113855361938, -1.3203418254852295, 1.416967749595642, 1.9548434019088745, 0.2977125346660614, -1.587518334388733, -0.3511905372142792, 1.8437000513076782], [1.178334355354309, 0.09746181964874268, 0.8947635293006897, -0.15067251026630402, 1.5602061748504639, -1.622567057609558, 0.4025844633579254, -0.6392581462860107, 0.3700789213180542, -2.122248888015747, 1.8175413608551025, -0.8539430499076843, 0.36084479093551636, 0.66881263256073, -1.5408833026885986, 0.4800371825695038, 1.0657182931900024, -1.2996044158935547, -0.9166600704193115, 0.06305674463510513, -1.134161114692688, -0.11674555391073227, 0.09557238221168518, 0.42927926778793335, -1.6351300477981567, -2.5265655517578125, 1.130279541015625, -1.256325125694275, 0.7653180360794067, 0.7959813475608826, -0.01911752112209797, 0.6925010085105896, 0.07578957825899124, -0.3440791666507721, -0.854925274848938, 0.3012988567352295, -0.6684240102767944, -0.6805287599563599, 0.6932090520858765, 2.022059679031372, 0.09952639043331146, 1.3580304384231567, -1.677384614944458, -0.2912450134754181, 0.5227134227752686, -0.24640332162380219, 0.29248714447021484, 0.19324709475040436, 0.06035742536187172, 1.217670202255249, 0.3681289851665497, 1.1824613809585571, 1.0328519344329834, 0.9975891709327698, 0.2606554627418518, -0.01656549982726574, -0.03103884495794773, 1.2645577192306519, -0.5643932223320007, 1.3363250494003296, 0.8694360852241516, -1.2944813966751099, -1.4502276182174683, 0.7741366624832153], [-1.520754337310791, -0.871219277381897, 0.4113866984844208, -1.4104453325271606, -1.1307682991027832, 0.2634314000606537, 0.7050605416297913, 0.0161051657050848, -0.033068932592868805, -0.2910422086715698, 1.1426023244857788, -1.4296513795852661, 1.9090436697006226, 1.2210825681686401, -1.3852639198303223, -0.29679322242736816, 1.6087517738342285, -0.07531822472810745, 1.2756119966506958, 0.12026890367269516, -0.5668033361434937, 0.3075031638145447, 0.9031727313995361, 0.21897956728935242, -2.074444532394409, -0.3368622958660126, 0.26767289638519287, 0.3724934756755829, 1.08989417552948, -0.12014393508434296, 1.36305570602417, -0.3352442979812622, 2.554081678390503, -0.8972657322883606, 0.5742945671081543, -0.3022089898586273, 0.061153843998909, 0.34576234221458435, -1.0454294681549072, 0.8596544861793518, -1.3916516304016113, -0.2184372842311859, -1.547429084777832, 0.18385939300060272, 0.8792438507080078, 1.137816309928894, -1.5027128458023071, -0.8405885100364685, 0.23091143369674683, 0.4763432443141937, -1.7373155355453491, 2.028190851211548, 1.481425166130066, 1.7093429565429688, -0.9850336313247681, 0.13744772970676422, 0.5503882169723511, -1.1872615814208984, -1.2615485191345215, 0.2879379093647003, -1.1749141216278076, -1.1318247318267822, 1.4369635581970215, 1.3319902420043945], [0.03589251637458801, -0.534801721572876, -0.5375293493270874, -0.3038325607776642, 0.4737768769264221, -1.163069486618042, 1.0837942361831665, -1.6264344453811646, 0.08072177320718765, 0.23723356425762177, -1.1323482990264893, -1.068989634513855, -0.4664206802845001, -2.3639278411865234, -0.5533840656280518, 0.692118763923645, 0.21309520304203033, 1.5170353651046753, 0.8812328577041626, -0.09880411624908447, -0.3761880397796631, 0.4291401207447052, 0.29268309473991394, -0.8344234228134155, -1.384504795074463, 0.46760356426239014, 0.3221864104270935, 0.2942505478858948, 2.061354160308838, 1.7000941038131714, -2.112169027328491, -1.624027132987976, 0.16291038691997528, 0.5078097581863403, 1.237693190574646, -0.5152403712272644, 0.27132871747016907, -2.1281065940856934, 0.5888624787330627, 1.8560619354248047, -1.2495856285095215, -2.0999794006347656, -0.5597153306007385, 0.620564877986908, -1.4300076961517334, -1.271589994430542, 0.8538525700569153, -0.022706259042024612, -0.20361368358135223, 1.3544117212295532, 0.7807524800300598, -1.3352757692337036, -2.218820810317993, 2.240586757659912, -2.1390042304992676, 0.16881605982780457, 0.8026483058929443, -0.4246038496494293, 0.1604602187871933, -0.49076998233795166, 0.30871450901031494, 1.0348773002624512, -0.3611587882041931, -1.3710037469863892], [-1.2667412757873535, 0.37895268201828003, 1.4926481246948242, 1.3976967334747314, 0.477620005607605, 2.0169477462768555, 2.002121686935425, -0.7344157695770264, 0.6735436916351318, 0.15025369822978973, 0.6969244480133057, -1.584004521369934, 0.5974863767623901, 2.1605124473571777, 1.7413338422775269, -0.6346700191497803, -0.22424070537090302, -2.970308780670166, 0.9008976817131042, 0.22584588825702667, -2.6396162509918213, -0.00297750998288393, -0.4367068409919739, -0.2785543203353882, -0.6147776246070862, 0.12291770428419113, 0.0757196694612503, 0.6189606189727783, -3.869598627090454, -0.31675007939338684, 2.2564735412597656, 0.49678507447242737, -0.11903439462184906, -0.15863870084285736, -0.7767614126205444, 1.2141824960708618, -0.12717323005199432, -1.5026456117630005, -1.071253776550293, -1.4054652452468872, 1.7053948640823364, 0.7261248826980591, 0.886508584022522, -0.057875923812389374, -0.9605187177658081, 1.7305576801300049, 0.10250823944807053, -1.5623843669891357, 0.8808778524398804, -0.08641917258501053, -1.073955774307251, 1.5931812524795532, 0.524556577205658, 1.0873748064041138, -0.1003008708357811, 2.2112903594970703, 0.048513263463974, -0.4746316969394684, 0.7089495658874512, 0.5960743427276611, -0.24941499531269073, -1.9330365657806396, 1.1046220064163208, -1.4436031579971313], [-1.7937061786651611, 0.4847831726074219, -1.5503876209259033, 1.5112581253051758, 0.6501671075820923, 2.1143691539764404, -0.7712725400924683, -0.34250789880752563, -1.626101016998291, 0.1926172971725464, 1.0752110481262207, -1.4885151386260986, 1.093657374382019, -1.658570408821106, -0.6867133378982544, -1.5394636392593384, -0.6316183805465698, 1.5483262538909912, 0.3733007311820984, -1.874891757965088, -0.13462240993976593, -0.4345807433128357, -2.666912794113159, 0.2843114137649536, 0.11944423615932465, 0.9404314756393433, -1.3270176649093628, -2.0290775299072266, 0.6217362880706787, -0.7120383977890015, 1.5822105407714844, 0.505138099193573, 0.36422693729400635, 0.07391498237848282, 0.2985501289367676, -0.09152654558420181, 0.7945848703384399, 1.0216630697250366, 0.6745365262031555, 0.14923006296157837, 0.15039633214473724, -0.023060597479343414, -0.34312620759010315, 0.2697451412677765, 1.1585496664047241, 1.0529046058654785, -0.3596336543560028, 0.0029972572810947895, -1.6246596574783325, -0.465170681476593, 0.895482063293457, -0.1893579661846161, 0.2978595197200775, 0.28276723623275757, -0.5308123826980591, 0.14274905622005463, -0.05805312842130661, -0.6426012516021729, 0.03546455129981041, -0.8528940081596375, 0.0015581226907670498, 0.3614261746406555, -0.6536251306533813, 0.2325325459241867], [-0.15085025131702423, 1.3374481201171875, -0.338779091835022, 0.11600243300199509, 1.0349611043930054, 0.5359549522399902, 0.4668174982070923, -1.5898805856704712, -1.0691372156143188, -0.234910249710083, -0.7096590399742126, 2.3577840328216553, 0.5862739086151123, -0.37695324420928955, 1.0691344738006592, 1.1707878112792969, -0.27505284547805786, 0.12091272324323654, -1.3496005535125732, -0.9247876405715942, 0.5581435561180115, -0.9333063960075378, -0.4349079132080078, -0.6413310170173645, -0.5318496227264404, -0.04900120943784714, 1.1729735136032104, -0.8474118709564209, -0.23884670436382294, 1.3097772598266602, 0.1790446639060974, -0.48870858550071716, 1.774937629699707, -1.1842992305755615, 0.15379242599010468, 1.0511107444763184, -1.318244218826294, 0.2766840159893036, -0.16198819875717163, -0.49261412024497986, -0.4800534248352051, 0.05840859189629555, 0.2987825870513916, -0.8768773078918457, -3.1861464977264404, 0.11662188172340393, 0.2965753674507141, 1.8873599767684937, 0.9498589634895325, 0.3912992775440216, -0.3129304051399231, 0.552308201789856, -0.581474781036377, 1.0890671014785767, 0.8065732717514038, 0.03631996735930443, -0.7442247271537781, 0.40003764629364014, 0.8955026268959045, 0.21839827299118042, 0.549909234046936, 1.6377689838409424, 1.707187294960022, 1.4015333652496338], [-0.9421288371086121, 0.21127551794052124, -0.0490521676838398, -0.20631733536720276, -0.5233270525932312, 0.6718683838844299, 0.7586541175842285, 0.20563748478889465, 0.13891705870628357, -0.062002114951610565, 0.06378860771656036, 0.6944798231124878, 1.6087322235107422, -0.4962059259414673, 1.7410995960235596, -1.388293981552124, -0.08542443811893463, 1.677740454673767, 0.9639711380004883, 0.8349743485450745, -0.7851974964141846, 0.34288302063941956, -0.7475882768630981, -2.4668679237365723, -0.6987162232398987, -0.7104406952857971, 0.6346479058265686, 0.7237027883529663, -0.3821415603160858, -1.6660360097885132, -0.8474980592727661, -0.47280848026275635, 1.193071722984314, -0.761776328086853, 0.027722332626581192, 0.11419183015823364, 0.2769135534763336, 0.8850811719894409, -1.4879111051559448, 0.8023156523704529, 1.3435425758361816, 0.6233696341514587, -0.15081274509429932, -0.46859702467918396, 0.5980350375175476, -1.0415375232696533, -1.6675108671188354, -1.1278226375579834, 1.0588210821151733, -0.5896696448326111, 0.30900564789772034, -0.9719666242599487, -0.6142474412918091, 1.793647050857544, -0.5484644174575806, -0.8005501627922058, -0.8361499309539795, 0.035295672714710236, -1.15566086769104, 0.3891490399837494, -0.5671920776367188, -0.5252138376235962, 0.48859521746635437, -0.2631162405014038], [-0.5536194443702698, -0.8921148180961609, 1.657463550567627, 0.9843774437904358, -0.3246970772743225, 0.7535154223442078, 0.8489563465118408, -0.04825791344046593, -0.1748715192079544, 0.28942811489105225, 0.3183049261569977, -0.00697017228230834, -0.43591541051864624, -0.6995000243186951, 0.8256911039352417, -0.7221396565437317, 0.851817786693573, -0.5388479828834534, 1.02785062789917, -0.658673882484436, 0.41221556067466736, 0.7127153277397156, 0.5539624094963074, 1.1647865772247314, 0.7568398118019104, 0.23919324576854706, -0.9761466383934021, -0.5084291696548462, -0.5231983065605164, 0.2649582028388977, -0.35359302163124084, -0.43793120980262756, -0.163896381855011, -0.9431186318397522, -2.13869571685791, 0.9252058863639832, 0.6395539045333862, -0.8591152429580688, -0.005247489549219608, 0.26091423630714417, 0.6943737864494324, -0.27251848578453064, 1.0858714580535889, -0.012545050121843815, 0.3518054187297821, 0.08683817833662033, -1.0239964723587036, 0.16853831708431244, -1.1976312398910522, -0.3374975025653839, -0.4501970112323761, -0.3275422751903534, -0.593845009803772, -0.4690588712692261, -1.1585215330123901, 0.05037061870098114, 0.003232122864574194, 0.713993489742279, 0.9951697587966919, 0.14608101546764374, -0.6884246468544006, 0.4317156672477722, 0.14750222861766815, -0.022974442690610886], [0.7703993320465088, -1.154704213142395, 1.8025891780853271, -1.2001646757125854, 1.2627031803131104, 1.172183871269226, -0.8068492412567139, -0.3709961771965027, -0.2426927536725998, 0.36739394068717957, 1.2823903560638428, -0.03315269574522972, -1.370460867881775, 0.36868658661842346, -1.0273221731185913, 1.2562637329101562, 0.15864652395248413, 0.011354015208780766, 1.3701298236846924, -0.3972017765045166, -0.9013113975524902, -1.4112831354141235, -0.8268406987190247, 0.9897984862327576, -1.8207626342773438, -0.2671891152858734, 1.2334556579589844, -1.4442386627197266, -0.5724688172340393, -2.68900728225708, 1.1236029863357544, 1.08427095413208, -1.1436339616775513, 0.5710005164146423, -0.4253588616847992, -0.19388502836227417, 1.5249344110488892, -0.11126280575990677, -1.0170750617980957, -1.0560166835784912, -0.22579610347747803, 0.3019857704639435, 1.7646827697753906, 0.27597126364707947, 0.47263601422309875, 0.3073006570339203, 1.296844720840454, -0.42022505402565, 0.8822240233421326, -1.021461009979248, -0.37628239393234253, -1.9360466003417969, 0.48974961042404175, -0.8554217219352722, 0.5152353048324585, 0.27391836047172546, -1.0354825258255005, -0.021883469074964523, 0.7415713667869568, -0.4410674571990967, 1.477927327156067, 1.5447571277618408, 0.3950302302837372, 0.490327924489975], [0.5273654460906982, 0.4168297350406647, 0.004853084683418274, 1.354175329208374, -1.0061123371124268, -0.6154966354370117, -1.2686898708343506, -1.772223949432373, 0.10913723707199097, -1.6539489030838013, 2.10919189453125, 0.7656986713409424, -0.8919095993041992, -1.223469614982605, -1.535150170326233, -0.17800192534923553, -0.5797951221466064, -0.35032010078430176, 2.3270275592803955, 0.9370866417884827, -0.586760401725769, -0.40101853013038635, 0.7714911699295044, 1.2862193584442139, -0.6500023007392883, 0.9942218065261841, 0.08602781593799591, 0.3114434480667114, -1.3719531297683716, 0.42679762840270996, 0.12991951406002045, -0.9267851114273071, 0.7610001564025879, -1.3184356689453125, 0.3319324254989624, 1.5191874504089355, 1.3856115341186523, -0.6564173698425293, -0.9166534543037415, 0.9458845853805542, -0.5616043210029602, 0.6147510409355164, 1.7663428783416748, 1.6958584785461426, -0.42604169249534607, -0.8142197132110596, -1.6100889444351196, 0.7282446026802063, -0.026223665103316307, 0.9221334457397461, -0.5203902721405029, -0.8984212279319763, -0.004866549279540777, -0.5601136088371277, 0.1161968931555748, -0.16948403418064117, 1.1101380586624146, -0.8398916125297546, -1.6419707536697388, 1.024733066558838, -0.20257706940174103, 0.10244403034448624, 0.3476872146129608, -1.1000393629074097], [0.4406794011592865, -0.5277665853500366, -1.1000914573669434, 0.5039530992507935, -0.7866429090499878, -0.4042850136756897, 0.5497255921363831, -0.6317446827888489, -0.3223510682582855, 0.48265430331230164, -1.1254979372024536, 0.8157751560211182, -0.9073025584220886, -0.15538787841796875, 0.11610785871744156, -0.038258183747529984, -1.9203158617019653, -1.2496364116668701, -0.9739928841590881, -0.9515048265457153, 0.0994018018245697, -0.15672004222869873, 0.3253297507762909, -0.09540974348783493, -0.5560065507888794, 0.8430463075637817, 0.8300796747207642, 0.7475616931915283, -0.417281836271286, -0.11413559317588806, 1.090842366218567, 1.0796235799789429, -0.5205891132354736, -0.5687420964241028, 0.19768700003623962, 0.6387044191360474, -1.8944278955459595, -0.32911717891693115, 0.5162392854690552, -0.6202659010887146, -1.404317021369934, 0.474971741437912, 0.1589745134115219, -0.1405481994152069, 1.4747930765151978, -1.666858434677124, 1.44465172290802, 0.28525295853614807, -0.5793704390525818, -0.18232613801956177, 0.15245486795902252, -1.10050368309021, 2.1663472652435303, 0.5003712773323059, 0.5045803189277649, 0.48876604437828064, 0.12513615190982819, -2.079768657684326, 0.3798317015171051, 1.2528873682022095, -0.45123401284217834, -1.5012279748916626, -0.2548982799053192, -0.2536660134792328], [-1.7755643129348755, 0.7214780449867249, 1.0252047777175903, -0.3415603041648865, -1.4652098417282104, 0.4670259356498718, 0.2788088321685791, -0.717691957950592, -0.833530068397522, 2.597729206085205, 1.6096205711364746, 1.4544035196304321, 0.5474348664283752, -1.1981675624847412, 1.2020660638809204, 0.456829309463501, -0.9255200624465942, 0.9234039783477783, 1.232597827911377, 0.27744829654693604, -0.07487618923187256, -0.7215651273727417, 0.0852295532822609, -0.24893391132354736, -1.9124778509140015, -1.6397647857666016, 0.9108067154884338, 0.2228320688009262, 0.298193097114563, -0.01595219597220421, -0.5751959681510925, 0.579867959022522, -0.748035192489624, -0.9225023984909058, -0.27000007033348083, 0.24891963601112366, -0.44688621163368225, 0.11419233679771423, 0.3015270531177521, -0.2822684943675995, 1.5568205118179321, 0.782041072845459, -1.437437891960144, 1.0842825174331665, 1.5194238424301147, -1.5261731147766113, -0.961025595664978, 0.842461884021759, 1.3027158975601196, -0.1478785276412964, 0.02983563020825386, -0.14964786171913147, -0.9446054100990295, -0.3146492838859558, 0.5423573851585388, -2.2202045917510986, 0.22838884592056274, -0.12381862848997116, -0.6747349500656128, -1.077295184135437, -0.597270667552948, -0.7447696924209595, -1.2142362594604492, -0.318209171295166], [1.1707745790481567, 1.747765302658081, -0.06191138178110123, 0.20839346945285797, -0.9541165828704834, 0.5459880828857422, -0.354824960231781, -0.47833451628685, 1.3034420013427734, -0.8390662670135498, 1.4634041786193848, 0.15521909296512604, 0.7974323034286499, 1.1255381107330322, 0.17247194051742554, -2.1010148525238037, -0.6172648668289185, 0.7594300508499146, -1.1922413110733032, -1.4809882640838623, 1.2771515846252441, -1.0345752239227295, 0.9986185431480408, 1.18020498752594, 1.6036275625228882, -0.5497681498527527, 2.3797717094421387, 0.332905650138855, 1.0164624452590942, -0.8967705965042114, 1.5933375358581543, 0.2576470971107483, 0.21117857098579407, 0.42567178606987, 0.697615385055542, -0.7650901079177856, 0.4885405898094177, -1.980393648147583, 0.11461851000785828, 0.760182797908783, -1.0329854488372803, -1.092569351196289, 0.6975357532501221, -0.642393171787262, -1.8597947359085083, -0.36479997634887695, 0.4899195432662964, 0.9159048199653625, -1.288217306137085, 1.9021939039230347, 0.5153645277023315, -0.25081881880760193, -0.6975582838058472, 2.4899742603302, -0.3342597782611847, -1.6650519371032715, -0.4336376488208771, 0.5896031856536865, -0.43026140332221985, -1.0846457481384277, 1.873660922050476, 0.48450908064842224, -0.8585372567176819, -1.6453149318695068], [0.39602038264274597, 1.888360857963562, 0.8181108832359314, 0.7394673824310303, 0.013080255128443241, 0.5714835524559021, 0.5147601962089539, -0.33751180768013, -0.7639632225036621, 1.7703145742416382, 0.23860208690166473, -0.2851324677467346, -1.036159634590149, 0.948210597038269, -1.5547356605529785, -1.027382254600525, 1.521604299545288, 0.6302538514137268, -1.3236253261566162, -0.9598629474639893, 1.2241029739379883, 0.5988196134567261, -1.8123751878738403, -0.11419782042503357, 2.36472225189209, 0.3223879039287567, 0.020588433369994164, -1.635466456413269, 1.4590998888015747, -0.391355037689209, 0.40694594383239746, 1.7337217330932617, 1.7743860483169556, -0.3865574300289154, -0.1935923844575882, 0.4589395821094513, 0.7645363211631775, 0.41656848788261414, -0.7042967081069946, -0.40226513147354126, 0.9379918575286865, -0.27412277460098267, -1.747711420059204, 0.59450364112854, 0.12254030257463455, -0.645021378993988, -1.3327691555023193, 0.19708006083965302, 0.578204333782196, 0.6505128741264343, -1.5022181272506714, -0.4065108001232147, 1.6949450969696045, 1.0805824995040894, -0.5312108993530273, -1.144751787185669, -0.5673092007637024, 0.1265099048614502, -0.3410644829273224, 1.497868537902832, 2.018702983856201, -1.1232824325561523, 0.12824290990829468, 1.594916582107544], [0.9871630668640137, 0.050243739038705826, -0.057999640703201294, 0.5007919669151306, -0.5203310251235962, -0.8601722121238708, 2.3726465702056885, -0.09775086492300034, 0.7382622957229614, -0.8795921802520752, -0.3474971652030945, 1.5333077907562256, -0.748871386051178, -1.3690812587738037, 0.029550381004810333, -1.4868407249450684, 0.6746847629547119, 0.7552395462989807, 0.20802626013755798, -0.4460436999797821, 0.010483566671609879, -0.2812767028808594, 0.06424205750226974, 1.8396592140197754, 0.6204358339309692, -0.28131842613220215, 0.03989831730723381, -1.1519887447357178, 0.31239065527915955, 0.26073023676872253, 0.07320217788219452, 0.9103527069091797, -1.5382144451141357, 0.5614517331123352, -1.4351657629013062, -1.4460725784301758, 0.8623130321502686, 0.6085916757583618, 0.8102776408195496, -0.2988329231739044, -1.3214218616485596, -0.05954788997769356, -1.7318015098571777, 1.7026655673980713, -0.9956127405166626, -0.2615058124065399, -0.012758039869368076, -0.5328888893127441, -0.016943441703915596, -0.11859128624200821, 0.3098408281803131, -1.1149228811264038, -1.213281512260437, 0.10315963625907898, -1.2824418544769287, -0.3768693506717682, 0.05954233556985855, -0.9732076525688171, -0.5779306292533875, -1.0630912780761719, -0.19937871396541595, -0.0012348570162430406, 0.6511722207069397, 0.5400415658950806], [-0.17521342635154724, 0.02591371349990368, 1.4332159757614136, -0.40320101380348206, -0.9089184999465942, 0.7396767139434814, -1.4203020334243774, -0.2659062147140503, -0.3290509879589081, -1.3392133712768555, -0.2819606065750122, 1.1917251348495483, -1.6967207193374634, -1.1258572340011597, 0.13533271849155426, -0.7653658986091614, 0.739641010761261, -0.7249017357826233, 1.069408893585205, -1.7226988077163696, -0.005569040775299072, 0.37960484623908997, 0.45655927062034607, 0.6890276670455933, -2.4520130157470703, -0.536979079246521, -1.1392171382904053, 0.029101157560944557, -1.5101063251495361, -0.3834945261478424, 0.9450950026512146, -1.347168207168579, 2.044182538986206, 0.716596782207489, -2.373884677886963, -1.785850167274475, -0.21181976795196533, 1.5007035732269287, 0.43811312317848206, -0.314614474773407, 2.4883124828338623, 0.121395543217659, 1.908174753189087, -0.6796833276748657, -1.387568712234497, 0.641834557056427, -0.048497699201107025, -0.36980509757995605, -0.44562023878097534, 1.09207022190094, 0.8266724348068237, 0.6211441159248352, -0.32196223735809326, 0.7131453156471252, -0.9731613993644714, 0.0357864573597908, 0.9952384233474731, 2.224414825439453, -1.5785996913909912, 0.660866916179657, -0.48902979493141174, 1.1709874868392944, -0.35807549953460693, 0.47928306460380554], [-0.2395252287387848, 0.43702107667922974, -1.1538130044937134, 0.7680521011352539, 0.8365068435668945, -0.6863071322441101, 1.2148044109344482, 2.0087504386901855, 2.295989990234375, 0.4002091586589813, -1.9926888942718506, -0.8139373064041138, -0.3411015272140503, 1.649279236793518, 0.6996890306472778, -1.4751150608062744, -0.23692263662815094, 0.529742419719696, -1.1134370565414429, 0.2949398159980774, -0.8644372820854187, 1.237035870552063, -0.764710545539856, -2.349489450454712, 1.0491381883621216, 0.21727798879146576, -0.7885699272155762, 0.21021167933940887, -0.6797834634780884, -0.9092360138893127, -0.8449587821960449, -1.4011281728744507, 1.1784754991531372, 0.5209747552871704, 0.29884958267211914, -1.358691930770874, -0.19643989205360413, 0.8195576071739197, 2.768209934234619, -1.0542941093444824, 0.6828732490539551, 0.2940552532672882, -0.03157973662018776, -0.31505388021469116, 0.365125447511673, 1.0882002115249634, 0.5120021104812622, 1.309913992881775, 0.26262030005455017, 0.15503916144371033, -1.0366497039794922, -0.6511070132255554, -0.5969434380531311, 0.315481960773468, 0.17746257781982422, 0.49078184366226196, 0.48266953229904175, -1.3034554719924927, -2.9122421741485596, 0.4151776432991028, -0.5208370685577393, -0.9664587378501892, -0.3608950078487396, 0.5958121418952942], [1.5925209522247314, 0.22768351435661316, 1.7404595613479614, 1.63130521774292, -0.12309211492538452, -1.2327533960342407, 1.1390608549118042, 1.063574194908142, 0.1729089319705963, -0.5565709471702576, 0.16862869262695312, 0.7694226503372192, -0.25702279806137085, -0.7388867735862732, 0.8367447853088379, 0.08315056562423706, 2.0291640758514404, 0.5059658288955688, -0.9929018020629883, -2.012190818786621, -0.6371460556983948, -0.32537227869033813, -0.8573000431060791, 0.654411792755127, -0.6921948790550232, -1.5573846101760864, 0.9593316912651062, 0.12187722325325012, 1.238857388496399, -0.2089943140745163, -1.4302133321762085, -0.8791835308074951, 0.3246760964393616, 0.4251948297023773, -1.4780423641204834, 1.2792736291885376, 1.2479958534240723, -1.505510926246643, -0.3852147161960602, -1.999686360359192, -0.7773454189300537, -0.3300668001174927, -0.27683064341545105, 0.35928666591644287, 1.9362655878067017, -0.901852011680603, -0.19170108437538147, -0.6439650654792786, -0.8260126709938049, -1.4178239107131958, -2.491016387939453, -0.9108437895774841, 2.206040382385254, 1.162973403930664, 0.2286013662815094, 0.12821489572525024, 1.685180425643921, 1.4325966835021973, 1.4048303365707397, 0.018172981217503548, -0.38674211502075195, 1.7429907321929932, 0.596625566482544, -0.7383976578712463], [0.35382381081581116, -0.048435550183057785, -0.46726536750793457, -0.9274324178695679, -0.8596230149269104, -0.4500972628593445, 0.7948214411735535, -1.0346076488494873, -0.08547689020633698, 1.4670690298080444, 0.534177303314209, -0.8265047669410706, -0.6420037150382996, -0.6684666275978088, -0.3311379551887512, -0.08847243338823318, 0.459053099155426, 0.04533781483769417, 1.358407974243164, 0.6310186982154846, 0.47740063071250916, 0.3644236624240875, 0.8129929304122925, -0.9457469582557678, 0.1768486052751541, -0.20044894516468048, 1.1912306547164917, -0.026506565511226654, -0.4103594720363617, -0.9388894438743591, -0.6236194968223572, 1.9335851669311523, -0.4104139506816864, -0.4300042986869812, -1.040690302848816, 0.29818645119667053, 1.3735151290893555, -0.38219398260116577, -1.4679261445999146, 1.4622435569763184, -0.6153167486190796, 0.5719494223594666, 1.1869193315505981, 0.7097099423408508, -1.0749703645706177, 0.6115738153457642, -0.7287915945053101, 0.7821680903434753, -0.4382864236831665, 0.4203983545303345, -0.26413244009017944, -1.1738272905349731, 1.0899736881256104, 1.8724371194839478, 0.8330483436584473, 0.729302704334259, -0.9383639693260193, 1.3552625179290771, -0.7684556245803833, -0.4089575409889221, -1.061942219734192, 1.4699666500091553, -0.20452119410037994, -0.2359458953142166], [-0.649873673915863, -2.350461006164551, 0.6513310074806213, 1.6816822290420532, 0.3139294683933258, -1.0303046703338623, -0.22378291189670563, -0.4119017720222473, -1.5956708192825317, -0.476949542760849, 0.4639703631401062, 0.16348473727703094, 0.453962504863739, -2.5660781860351562, -2.0938022136688232, -0.7188193202018738, 1.8788479566574097, 1.3927026987075806, 0.49224525690078735, 0.7308105230331421, 1.212660789489746, -1.1814069747924805, 0.6837201714515686, 0.5258036255836487, 0.05695436894893646, -1.7099577188491821, -1.7124441862106323, 1.2589980363845825, -1.0561319589614868, -1.422203779220581, 1.7367719411849976, 1.8650130033493042, 0.550075113773346, -0.1998746693134308, -0.708523154258728, 0.9021920561790466, 0.5685099363327026, 0.9463979005813599, 1.2971690893173218, 0.0491018071770668, -1.2766470909118652, 0.6783608794212341, 2.2040984630584717, 1.0874437093734741, 0.08568231761455536, 1.745087742805481, -1.0050514936447144, 0.5102793574333191, -0.36376190185546875, 0.054148875176906586, 1.4486407041549683, 0.3666965365409851, 1.2023389339447021, 0.8985658288002014, -0.8364112973213196, -0.7589685916900635, 1.4812431335449219, -2.2536771297454834, -0.2398632913827896, -1.872563362121582, 0.6721929311752319, -0.21018418669700623, -0.1361534297466278, -0.46655648946762085], [0.8916932940483093, 1.3799060583114624, -0.7615683078765869, -0.21479815244674683, -0.7435439825057983, 0.25134992599487305, -0.2170533835887909, 0.3378148078918457, -1.3939213752746582, -2.0396487712860107, -0.3676799535751343, 1.7761088609695435, 2.9965853691101074, -0.9876382350921631, 0.12987729907035828, -1.0405007600784302, 1.5412801504135132, 0.44268450140953064, 1.1389760971069336, -0.8495386242866516, -1.0988647937774658, 0.8751038312911987, 0.30740049481391907, -1.576246738433838, -3.613847255706787, -0.5948628783226013, 0.5898030996322632, -1.0797410011291504, 0.8679891228675842, -0.1216980367898941, 1.229672908782959, -0.2568223774433136, -0.987410306930542, 0.002701861085370183, 0.8997814655303955, -1.3583624362945557, -0.10876161605119705, -1.2785073518753052, 0.5706052184104919, 1.326795220375061, -1.8940045833587646, 1.2233291864395142, 0.6742077469825745, 0.08283664286136627, 0.544854462146759, 0.6554305553436279, 0.3063605725765228, 1.762709140777588, -0.6977043151855469, 0.4973798096179962, 0.6559333205223083, 1.4660756587982178, 0.8462828397750854, 0.08948718011379242, 0.5816651582717896, -1.1822680234909058, 0.5812123417854309, 1.4028314352035522, -0.2528386116027832, -0.10244844853878021, 0.6124419569969177, 0.7190463542938232, -1.5305384397506714, -0.382306843996048], [0.80617356300354, 2.0198023319244385, -0.09424193203449249, 0.44396132230758667, -1.3582361936569214, -1.5945125818252563, 0.780509889125824, 0.8160637617111206, 1.110101580619812, -2.1381518840789795, -1.5930957794189453, -0.36395010352134705, 1.0708513259887695, 0.5127445459365845, -0.32712510228157043, 1.4486875534057617, 0.6972970366477966, -0.1343507319688797, 1.8117477893829346, -0.3268944323062897, 0.5738271474838257, 0.5555520057678223, 0.9021005630493164, -0.3926088511943817, -0.6176581382751465, -1.0237727165222168, 0.34776321053504944, 1.437494158744812, 1.6662973165512085, -1.2537705898284912, -0.8752217888832092, -0.43469640612602234, 2.373112201690674, 0.5334230661392212, 0.7450670003890991, 2.406085252761841, -0.9110099077224731, -0.09516392648220062, -0.6296698451042175, -1.832626223564148, 1.015920877456665, 0.6734617948532104, -0.04343197122216225, -1.3998844623565674, 1.4717164039611816, -1.7745943069458008, 0.38109663128852844, -0.12124863266944885, 0.40507009625434875, -1.28092622756958, -0.45974478125572205, -0.17493264377117157, -1.4113883972167969, -1.5651202201843262, 0.3417605757713318, -0.49907565116882324, -1.1383304595947266, -0.9432628750801086, -1.1977815628051758, -2.105844020843506, -0.42818573117256165, -0.3211059868335724, 0.2681092321872711, 0.540526807308197], [-0.20575502514839172, 1.073033332824707, 0.9853108525276184, -0.16084769368171692, 0.45208001136779785, 0.3914821147918701, -0.38789522647857666, -0.2976965308189392, 0.08735296875238419, 0.20440001785755157, -0.09550777077674866, -0.2906106114387512, 0.14880964159965515, -0.7194771766662598, -0.8222537040710449, 0.9679376482963562, -1.1919682025909424, -0.29947689175605774, 0.15355926752090454, -1.9376636743545532, -0.6915652751922607, 0.8179154992103577, -1.3561527729034424, -1.042943000793457, 0.10223814100027084, 0.42635756731033325, -1.1183637380599976, 0.8764490485191345, -1.4433139562606812, -0.35730624198913574, -0.7488413453102112, -2.936696767807007, 0.42992252111434937, -0.3518432378768921, 1.0702561140060425, -0.5281707048416138, 1.0502898693084717, -0.44035500288009644, 1.5192240476608276, -1.225943684577942, 0.4066735804080963, 1.4087767601013184, -0.7657939791679382, 0.19478456676006317, 1.3983118534088135, -2.5844154357910156, -0.8284091949462891, -0.16554778814315796, 0.8563281297683716, -1.8022685050964355, 0.6244381666183472, 0.050677962601184845, 0.32061129808425903, -1.2294148206710815, 1.5131115913391113, -0.09989289194345474, -0.5804973840713501, 0.5314008593559265, 0.1317308247089386, -0.08393498510122299, -0.939416229724884, 1.0097899436950684, 1.1804004907608032, 2.407571792602539], [0.3367951214313507, 0.37362387776374817, 0.7327959537506104, 1.2854204177856445, -0.24326112866401672, 1.5225117206573486, -0.48496145009994507, 0.009960386902093887, -0.015256572514772415, 1.0810751914978027, 2.810195207595825, -0.3102325201034546, -0.6922010183334351, 1.2558938264846802, -0.048691220581531525, 0.8810455203056335, -0.21251583099365234, 0.2108066976070404, 1.5975335836410522, 1.270024299621582, -0.16166791319847107, 0.4973587393760681, 0.8017058968544006, 0.36931630969047546, -0.18443375825881958, -1.468299388885498, 1.6299108266830444, -0.380752295255661, -1.6611220836639404, 0.5351939797401428, 0.2741016149520874, 0.26299524307250977, 0.447800874710083, 0.5864551663398743, -0.8574181795120239, -0.10706380009651184, 1.149219274520874, 0.1414996236562729, -2.2812302112579346, 0.4438157081604004, -1.0594251155853271, 2.0435314178466797, -0.12571686506271362, -0.33617329597473145, 1.1155387163162231, 1.0161813497543335, 0.05464738979935646, -0.6891336441040039, 0.8143430948257446, -0.16009803116321564, -0.7499780654907227, 1.0320807695388794, -0.4988504946231842, -0.66799396276474, 1.4160985946655273, -0.15389186143875122, -0.5364140868186951, -0.01221142616122961, 1.6119272708892822, 0.47646957635879517, -0.3648603856563568, -0.42096689343452454, 0.42307886481285095, -0.8727368712425232], [2.4851105213165283, -0.23304995894432068, -0.2737446427345276, -0.3609274923801422, -0.6568212509155273, -0.03979623690247536, 0.37253910303115845, 0.979240894317627, 1.9331573247909546, 0.13262887299060822, 0.2069249004125595, 1.3955060243606567, -0.9741743803024292, -0.6586705446243286, 0.7717472314834595, -0.9326249957084656, -0.5832547545433044, -0.7167041301727295, -0.7358180284500122, 1.438892126083374, 0.37462398409843445, 0.7582894563674927, -0.5748087763786316, 1.053023338317871, 0.010170008055865765, -0.0053163631819188595, 1.3905011415481567, -0.16136999428272247, 0.6864089965820312, -0.14381763339042664, -0.10099690407514572, 0.5356292128562927, 0.7021446824073792, -0.567532479763031, -0.7974436283111572, 1.580623745918274, 0.26507776975631714, 0.09606286883354187, -0.9646065831184387, 0.5760431885719299, 2.18685245513916, -0.9798292517662048, -1.2293330430984497, -0.31325972080230713, -0.2490004450082779, 1.0994737148284912, -1.2401459217071533, -0.8234192728996277, 1.6779074668884277, 1.7858887910842896, -0.2322203367948532, -0.4411173164844513, 0.7843673229217529, 0.9383001923561096, 1.3503072261810303, -0.46818122267723083, -1.8165738582611084, -1.6067839860916138, 1.4605636596679688, -0.11969175934791565, -1.2952265739440918, -2.3352608680725098, 0.7858365178108215, 0.6935462355613708], [0.5761824250221252, -0.465471476316452, -0.3149130940437317, -0.13526861369609833, -0.14242324233055115, 0.2598210573196411, 0.4978697896003723, 0.684172511100769, -0.04664667323231697, -0.7475659251213074, -0.1913215070962906, -2.2836086750030518, 0.4157745838165283, 0.49695682525634766, 0.16657684743404388, 0.23260678350925446, 0.42731529474258423, -1.6779215335845947, 1.0676405429840088, -0.30101266503334045, 0.06836885958909988, 0.46753570437431335, 0.258848637342453, -0.14797112345695496, -0.8148640990257263, 0.9040833115577698, 0.259773850440979, -0.5227262377738953, -1.3157964944839478, 0.172922283411026, -1.5922621488571167, -0.9639129638671875, 0.9276937246322632, 0.5162739157676697, 0.18169869482517242, -0.680675745010376, -0.6132815480232239, -0.8270045518875122, -1.0430445671081543, 0.10939548909664154, 0.39277559518814087, -1.5791094303131104, -0.6529937386512756, -0.5757043957710266, 0.17828597128391266, -0.2647648751735687, -1.0387290716171265, -1.1715904474258423, -0.29404523968696594, 0.8512457013130188, -2.7700355052948, 1.0020451545715332, 0.7685035467147827, -0.1734887659549713, 0.829334557056427, 0.963104784488678, -0.7502920627593994, -0.634301483631134, 0.3286533057689667, -1.4895395040512085, 2.6013131141662598, -1.8788495063781738, -1.079627275466919, -1.3565446138381958], [0.09016139805316925, -0.6520211696624756, 0.5636163949966431, -0.20429271459579468, 0.39824095368385315, -0.35167691111564636, -0.8582764863967896, -0.8388108611106873, 0.2350880354642868, 0.01634046994149685, -0.3703811466693878, -1.8359308242797852, -0.5201085805892944, -0.29776531457901, -0.8886498808860779, -0.19376696646213531, -1.0818431377410889, -0.2200913280248642, 0.31272903084754944, -2.272347927093506, 1.7266477346420288, -1.4839324951171875, 0.6633639931678772, -1.3846887350082397, -0.2289610505104065, -0.5510251522064209, -0.13169707357883453, 0.8910763263702393, -1.5334315299987793, 1.6082979440689087, 1.1577304601669312, -0.08710690587759018, -1.1373463869094849, 0.8005186915397644, -0.7731593251228333, 0.2485567331314087, -1.4235682487487793, 1.8323984146118164, -0.5212709903717041, 0.6985001564025879, 0.7839440703392029, 1.506514549255371, 0.9547235369682312, 0.9267700910568237, 0.1288309395313263, 0.23190937936306, 0.5145323872566223, 0.29878929257392883, -0.8079354763031006, -1.680747628211975, 0.07886837422847748, -0.6733609437942505, 0.6699373722076416, 0.46599817276000977, -0.09088771045207977, -0.07834279537200928, 0.8047268986701965, -0.5798014998435974, 0.8159964084625244, 0.0043536401353776455, 1.2075886726379395, 1.5882965326309204, -0.9944143295288086, 0.3553904891014099], [0.29675376415252686, -0.6871514320373535, -0.23114176094532013, 0.09277796745300293, -0.7512926459312439, 1.209243655204773, -0.9639649391174316, 0.5113198161125183, 0.9854325652122498, 0.3660125434398651, 0.7422646284103394, -0.049511831253767014, -0.684066891670227, -0.11998593807220459, -0.22193290293216705, 0.5988878607749939, -1.3971517086029053, -0.20621679723262787, 1.83004629611969, -0.802536129951477, -0.43692830204963684, 0.5968037843704224, -1.6008261442184448, -1.811439037322998, -1.8568062782287598, 0.9008644819259644, 0.6367197036743164, 0.7618365287780762, -0.9669521450996399, -0.9762251973152161, 0.2207721471786499, -0.38563406467437744, -1.000707983970642, -0.8354752063751221, 1.0885335206985474, 0.31588441133499146, 1.3010084629058838, 0.4426783323287964, -1.6244828701019287, 0.05141378939151764, -0.7804995179176331, -0.023097924888134003, 0.6635017991065979, 1.7463223934173584, -1.0891380310058594, -2.868089437484741, 0.6025259494781494, -0.2786225378513336, 0.09920958429574966, 0.4633753001689911, 1.4296811819076538, 0.24313171207904816, -2.440908193588257, -0.8871859312057495, 0.7097509503364563, -0.712567150592804, -0.6193283200263977, 0.02210298180580139, -0.8301578164100647, 0.5741461515426636, 0.6032026410102844, -0.2690867483615875, -0.8804200291633606, 0.38143089413642883], [-0.36910104751586914, 0.9382685422897339, 1.5229393243789673, 0.4319570064544678, 0.5730421543121338, 0.43549880385398865, 1.403210163116455, -0.2543042302131653, 0.6164454221725464, -0.36650219559669495, 0.26236602663993835, 0.7301288843154907, 1.503230094909668, 1.7264792919158936, 2.032916784286499, -1.8259540796279907, 1.491958498954773, 1.1537336111068726, 0.8088562488555908, 0.19218674302101135, 0.08458293229341507, 0.5047852993011475, -0.18914653360843658, -0.9515162110328674, -0.4252853989601135, -0.8776463270187378, 0.0013499159831553698, -1.0516901016235352, -0.07865442335605621, 0.34576302766799927, 0.6773317456245422, -0.8243990540504456, -0.5257133841514587, 2.1958513259887695, 0.5197786092758179, -1.0758092403411865, 1.5493327379226685, 0.5979506969451904, -0.31996527314186096, 0.9639144539833069, 1.6623032093048096, 0.2975202798843384, -0.16097116470336914, -0.10407248139381409, -1.0442930459976196, 0.4042244851589203, 1.7407078742980957, -0.29823046922683716, -0.49962371587753296, -0.029140105471014977, -0.6113385558128357, 0.2518521845340729, -0.3166223168373108, 0.06296946108341217, 0.4255541265010834, -1.6657061576843262, -0.9536203742027283, 0.14793872833251953, -0.2616773843765259, -0.5138007402420044, -0.03885411098599434, -2.989269256591797, -1.1949269771575928, 1.1480869054794312], [-0.7258137464523315, 2.3384437561035156, -1.1139522790908813, 0.5071627497673035, 0.9528759717941284, -1.3042794466018677, -0.04524330422282219, 0.7548374533653259, -1.426151990890503, -0.5221980214118958, 0.0687742754817009, -0.4091811776161194, -0.7894862294197083, -0.10573383420705795, -2.2000420093536377, -0.9592289328575134, -0.2908647656440735, 0.1668848693370819, -1.1804956197738647, -0.7970715165138245, 2.1879851818084717, 1.333516240119934, -1.4480375051498413, -0.24298901855945587, -0.7222221493721008, -1.8095788955688477, 0.18176044523715973, -0.010149748995900154, 0.39078375697135925, 1.1783647537231445, -0.15401868522167206, 0.14066234230995178, 0.90229332447052, 0.8210976123809814, 0.18354561924934387, 0.19779251515865326, 0.566779613494873, 0.09062644094228745, -0.9247626662254333, -0.7498174905776978, -1.0647832155227661, 1.2556644678115845, 0.8804863095283508, -0.4347957372665405, -0.8021714091300964, 0.6183408498764038, -0.6258617043495178, -1.051406741142273, 0.019194763153791428, 0.5414417386054993, -0.5174789428710938, -1.1991697549819946, -0.4011470377445221, -0.07051810622215271, 0.38256436586380005, -0.6872785687446594, -0.5200284123420715, 0.6169628500938416, -0.10674901306629181, -0.40767037868499756, -0.1043974831700325, -0.1815536767244339, 0.5543122291564941, -0.21824002265930176], [-0.008466526865959167, -1.2023588418960571, -0.11287177354097366, -0.5092609524726868, 0.35695910453796387, -0.20900145173072815, -0.13839606940746307, -0.739508867263794, 1.5071947574615479, 0.6867622137069702, -0.39023229479789734, 1.0566771030426025, 0.5132160186767578, -1.2858498096466064, -0.46662819385528564, -0.5369202494621277, 0.6185011267662048, 1.0127593278884888, 0.5737993717193604, 0.5054656267166138, -0.7726224064826965, -0.49335217475891113, -1.2019988298416138, -0.8840818405151367, -0.5285718441009521, 0.2562335431575775, -0.7473153471946716, -0.14820244908332825, -0.20004302263259888, 0.09084969758987427, 0.13130471110343933, -0.8461037278175354, -1.6918686628341675, 1.7138428688049316, -0.009142548777163029, -2.1812217235565186, -0.18566839396953583, 0.12557286024093628, 1.2149540185928345, -0.395541250705719, -0.5823513269424438, -0.24073968827724457, 0.954675555229187, 0.8555984497070312, -0.3451310098171234, 1.23061203956604, 0.6482586860656738, 0.7941858172416687, 0.5099152326583862, -0.45600563287734985, -1.2987313270568848, -0.45894110202789307, 0.15798935294151306, -0.41628497838974, -0.13925226032733917, 0.6277949213981628, 0.10524434596300125, 1.4859404563903809, -0.5035930275917053, 0.6845011115074158, 1.6215455532073975, 0.8597538471221924, -0.09758513420820236, 0.09299011528491974], [-0.29207465052604675, -0.32915306091308594, 1.5383447408676147, 1.161810278892517, -0.07353314012289047, 0.8921197652816772, 0.7668967843055725, -0.21191374957561493, 0.838819682598114, -0.4620848000049591, 0.4687751531600952, -0.33323588967323303, -1.079163908958435, -0.1917765736579895, -0.46762028336524963, -0.2643558382987976, 0.6344428658485413, 0.3676208555698395, -1.0319594144821167, 0.8383339643478394, 1.3585073947906494, -1.5209705829620361, -1.097334861755371, -0.7642279267311096, 0.28532862663269043, 1.0402671098709106, 1.7703524827957153, 0.8403049111366272, 1.400381088256836, 1.443814992904663, -0.5486222505569458, -0.3849841058254242, 0.7671477794647217, -0.5987773537635803, 0.370755136013031, 1.8751031160354614, -0.10339901596307755, -0.4986170828342438, -1.5090887546539307, 0.9740599989891052, 0.1429632604122162, 1.031691551208496, 0.18187421560287476, 1.14646577835083, 1.3174902200698853, -1.212830662727356, -0.25706082582473755, 0.7732433676719666, -0.9794110655784607, 0.45750853419303894, 1.2326079607009888, -0.3021716773509979, -0.4502216577529907, 1.321901559829712, 0.4666253924369812, -1.2275201082229614, 2.7488176822662354, 2.0130574703216553, 0.7731043696403503, -0.39214009046554565, -0.22249162197113037, 0.4767988622188568, 0.04269244521856308, -0.1741328239440918], [1.0437978506088257, -1.831546425819397, -0.4904986619949341, 0.3799683749675751, -0.31698447465896606, 0.5625994801521301, 1.067919373512268, -0.47732290625572205, 0.2617863416671753, -0.34917324781417847, 0.24202008545398712, -2.0819895267486572, 0.7044855356216431, 1.093665361404419, -0.14874832332134247, 1.9339194297790527, -1.3316888809204102, 0.38580799102783203, -1.1775176525115967, -0.9043423533439636, 0.1368655413389206, -0.729552149772644, 0.4665675163269043, -1.2854143381118774, -1.0936673879623413, 0.4167667031288147, -0.9615296125411987, -0.1522841900587082, 2.3253726959228516, 0.2957860827445984, 0.7878375053405762, 0.13069301843643188, -1.2738035917282104, -1.1371170282363892, -1.0050715208053589, -0.3676133155822754, 0.4951588213443756, 1.0935896635055542, -0.3702741861343384, -1.0647728443145752, -1.328792929649353, 0.12264683097600937, 0.2892875373363495, 0.2728947699069977, -0.05825313180685043, -0.938843309879303, -1.9211649894714355, -1.7839665412902832, 0.3127953112125397, -0.24273870885372162, 0.9477908611297607, -1.5222324132919312, 1.580995798110962, 0.0194498710334301, 0.5189266800880432, -0.2582647204399109, 1.4486092329025269, -1.6221128702163696, -0.4551128149032593, -0.46642354130744934, 0.32678061723709106, 1.3448048830032349, -1.6475316286087036, 0.9493811130523682], [-0.5503585934638977, -0.4012291431427002, -0.08625612407922745, -0.5998290777206421, -2.9350550174713135, -0.9300801753997803, 0.05375174432992935, -0.7239165306091309, 1.3775203227996826, -0.34917065501213074, 0.8618144392967224, 0.13932420313358307, 1.4179298877716064, 0.48041731119155884, 0.46943211555480957, -0.6478312015533447, -1.2337720394134521, 0.5364232063293457, -1.2387714385986328, 0.3926669657230377, -0.3836907744407654, -1.2696353197097778, 0.936148464679718, 1.0387868881225586, 1.413804054260254, -0.2561722993850708, 1.0367323160171509, 0.8814057111740112, -0.18485552072525024, 0.8299638628959656, -1.1245770454406738, -2.099573850631714, 0.2455616295337677, 0.35682857036590576, -0.31588658690452576, 1.7985938787460327, -1.5713145732879639, 1.0138347148895264, -1.2896649837493896, 0.5058704018592834, 1.148084044456482, -0.42796531319618225, -0.2735729515552521, -0.6525508165359497, -0.7785342335700989, -0.5682379007339478, -1.3190937042236328, -0.4830239713191986, 0.30450302362442017, -0.7678889632225037, -0.9650530815124512, -0.4744543731212616, 1.4747878313064575, -0.475546270608902, 0.02851453423500061, -1.4014511108398438, 0.8941817283630371, 1.0531882047653198, -0.7818951606750488, 0.27522364258766174, -0.5149273872375488, -0.3724578022956848, -1.0421956777572632, -0.05335921794176102], [0.8609248399734497, 1.794342279434204, 0.4707162082195282, -0.6333749890327454, 1.2853938341140747, -1.2981133460998535, 0.37570932507514954, 1.21101713180542, 0.029220597818493843, -0.810333788394928, -0.9760357141494751, -0.798748254776001, 0.3119587302207947, -1.8477520942687988, -0.16848266124725342, 1.2758843898773193, -0.1671065241098404, 1.0627714395523071, 1.323836088180542, 0.22197583317756653, -1.4564493894577026, 0.12150182574987411, 0.06312507390975952, 0.43492695689201355, 0.1272861510515213, 0.15847544372081757, -0.36386823654174805, 0.7822441458702087, -1.2755588293075562, 0.552393913269043, 0.6344670057296753, -1.0220117568969727, -0.8718246817588806, -0.26898080110549927, -0.40339571237564087, 0.6031976938247681, -0.8827006220817566, -1.199141263961792, 0.5142777562141418, 0.2625819146633148, 0.2925494909286499, 0.7732861042022705, 1.1475480794906616, 0.651226818561554, -1.5593818426132202, 0.9044061899185181, 0.34594038128852844, 1.2665770053863525, 0.8253558278083801, 0.13702771067619324, 1.877793788909912, -0.49502232670783997, 0.31059181690216064, 0.14148658514022827, 0.16119611263275146, -0.7632946372032166, 1.7801593542099, 1.4828789234161377, 1.127255916595459, 0.6501726508140564, -1.542534351348877, 0.7010446786880493, 0.4958755373954773, 0.7519968152046204], [0.10047460347414017, -2.1027097702026367, -1.0075879096984863, 0.3767651915550232, 1.9417526721954346, -0.2686169147491455, 0.36540326476097107, 0.49482110142707825, 0.9701249003410339, 2.057283878326416, 0.05798061564564705, 0.3220326900482178, 0.17012228071689606, -0.7958806753158569, -0.7259679436683655, 1.7211673259735107, 0.5308700799942017, 1.410670280456543, -1.8638824224472046, -0.9854518175125122, 0.3981161117553711, 0.35358184576034546, -0.9466322064399719, -1.3717893362045288, -0.45632609724998474, 0.7257893085479736, 0.6261858344078064, -0.3213030993938446, -0.5492926836013794, -1.5775591135025024, -0.15749382972717285, 1.4491665363311768, -0.6244842410087585, -1.0710816383361816, 0.018388234078884125, -0.16423630714416504, 1.325161337852478, 0.08981315046548843, 0.940802276134491, 0.09948589652776718, -0.45375895500183105, 0.521395742893219, 0.9969616532325745, -0.6715721487998962, 2.5552961826324463, 0.34352877736091614, 0.5474684834480286, -1.6224641799926758, -1.8363932371139526, -1.4744096994400024, 0.4691130518913269, -0.43458664417266846, 0.6253999471664429, 0.6723576784133911, 0.826631486415863, 1.9127172231674194, -1.473604679107666, 0.11854386329650879, -0.6887410283088684, 1.605774998664856, -0.8608418107032776, -1.4552404880523682, 0.41548967361450195, -0.8295629620552063], [0.445801317691803, 0.7984746694564819, 2.15828537940979, 0.6898969411849976, 0.6135567426681519, -1.0810590982437134, 2.145179033279419, 0.2930920720100403, -0.7666161060333252, 1.3303934335708618, 0.7674082517623901, -0.14305397868156433, 1.3078144788742065, -1.2546217441558838, -2.960994243621826, 2.055877447128296, 0.9809249639511108, -2.541440963745117, -1.5008502006530762, -1.8590490818023682, -1.0887621641159058, -0.2462587058544159, 1.048779010772705, -0.665558397769928, -0.021649381145834923, 0.15880286693572998, 0.307345986366272, -1.448846697807312, -0.006225985009223223, -0.8508313894271851, -0.19954417645931244, 1.3318305015563965, 0.43874722719192505, 0.6285277009010315, -1.6157549619674683, 0.25075340270996094, 0.14158135652542114, -0.7735543847084045, -1.1447796821594238, 0.5792700052261353, -1.2358778715133667, 1.2495067119598389, 0.247450053691864, 0.7085738182067871, 0.33395883440971375, 0.5305211544036865, -0.09845046699047089, -1.7909756898880005, 1.0859596729278564, 1.0269982814788818, 0.8350851535797119, 0.9893177151679993, 0.0351538360118866, -1.0756654739379883, -0.719990074634552, 0.30391085147857666, -1.2499366998672485, 0.563417375087738, -0.5983797907829285, 0.2741391956806183, 0.44511520862579346, 0.7007603645324707, -1.009387731552124, -0.560650646686554], [0.3741902709007263, 0.7921923398971558, 1.5047153234481812, -1.9624955654144287, -1.888005018234253, 2.3581316471099854, -1.6800023317337036, -0.8100715279579163, -0.5127495527267456, 0.3011673092842102, -1.197395920753479, -0.024103276431560516, 1.2003048658370972, -0.969651460647583, -0.2258254438638687, -0.027409648522734642, -0.10758362710475922, -0.5944230556488037, -1.2395424842834473, 1.077096700668335, -1.1993926763534546, 0.042163096368312836, 0.5810940265655518, 1.000137209892273, 1.438028335571289, 0.7898953557014465, 2.0513722896575928, -1.405206561088562, -2.1307075023651123, -0.35224491357803345, -1.2574634552001953, 0.7799410820007324, -1.2088061571121216, 0.08170285075902939, -0.5112879872322083, -0.07441785931587219, 0.24042221903800964, -0.6561019420623779, 1.4814964532852173, -1.351260781288147, -0.05105997622013092, -0.014030235819518566, -0.4463716447353363, -1.2401316165924072, -0.7934492230415344, 0.38750171661376953, 0.6942976117134094, 1.2957693338394165, 2.366316080093384, -0.5019664168357849, 0.750011682510376, 0.6254129409790039, 0.27134761214256287, 1.1693979501724243, -0.1068882942199707, -0.45236843824386597, 0.5769310593605042, -0.43963587284088135, 0.045369744300842285, 1.3788971900939941, 0.40306782722473145, -0.4009592533111572, -0.16242706775665283, -0.07532572001218796], [-1.7663179636001587, 0.8135542273521423, 0.17214645445346832, -0.21858027577400208, 0.06631185114383698, 0.2694816291332245, -1.0517351627349854, 0.15395349264144897, -0.031038012355566025, -0.7736371755599976, -0.8046246767044067, -1.2999110221862793, 0.3521416485309601, 1.3470433950424194, -0.2934052050113678, 1.2149043083190918, -0.9936333894729614, 0.5217082500457764, -0.07974118739366531, -1.3696364164352417, -1.7653559446334839, -0.8032127618789673, 0.20122404396533966, 0.5568020939826965, 1.3022512197494507, 0.7534324526786804, -0.05377905070781708, -1.1054857969284058, 2.430412530899048, -1.1272974014282227, -0.16226314008235931, -0.5600013136863708, -0.7407046556472778, -0.8385393023490906, 0.8491891026496887, 0.6976398825645447, 0.45387253165245056, 1.0108915567398071, 2.211345672607422, 0.16556066274642944, -1.5070180892944336, 0.29751861095428467, 0.6748963594436646, -1.158149003982544, 0.27392613887786865, -0.15023337304592133, 1.4018691778182983, 2.0626769065856934, -0.6427334547042847, -0.7831434607505798, -1.4505608081817627, 0.736176609992981, 0.40642109513282776, -1.2670848369598389, -0.2916077971458435, 1.417603611946106, -1.237026333808899, -0.5957457423210144, -0.36993408203125, -0.2466854304075241, 0.9641748070716858, 1.19084894657135, 0.16958409547805786, -1.257203459739685], [-0.2896362245082855, 0.688933253288269, -0.3016856908798218, 0.5343277454376221, 0.7686814069747925, 1.0608458518981934, 0.4097442030906677, -0.4637724757194519, 0.18790143728256226, 0.022886430844664574, 0.08460517972707748, 0.30424031615257263, 1.0646440982818604, -0.15232017636299133, 0.7267080545425415, -2.0532355308532715, 1.7005962133407593, -0.04676403850317001, -1.0634746551513672, -0.41194459795951843, 0.15469969809055328, -0.3450770974159241, -1.0264068841934204, 1.6359366178512573, -3.040894031524658, 0.2816021144390106, -1.1058692932128906, -1.4312807321548462, 1.2640056610107422, -1.1424928903579712, 0.4511730372905731, 0.38336560130119324, 0.8769956827163696, -0.24550512433052063, -0.06326275318861008, 1.210568904876709, 0.5486209392547607, -0.4016038179397583, -0.565410852432251, -0.8998164534568787, -0.5799548625946045, -0.7359402179718018, -0.716286838054657, -0.17099647223949432, -1.8860877752304077, -0.09721237421035767, -1.2527762651443481, -0.4375908076763153, -0.21473346650600433, 0.0979839414358139, -1.429101824760437, -0.43135911226272583, -0.7715576887130737, -1.0096006393432617, -1.0367110967636108, -0.9306109547615051, 0.1927674263715744, 1.1588072776794434, 0.4484075903892517, -0.3467924892902374, 1.0293655395507812, -0.7129291296005249, -1.8510260581970215, -0.2628805935382843], [-0.8216459155082703, -0.03632602468132973, 0.5780749917030334, -0.9435312747955322, 0.384798139333725, 1.5360969305038452, -0.3681522011756897, 0.24687820672988892, -0.7639066576957703, 1.585283637046814, 0.004497214686125517, -1.7665189504623413, 0.04400840029120445, 0.36259323358535767, -0.173222154378891, -0.5908711552619934, -0.0726127177476883, -0.8515973687171936, 1.395445704460144, 0.6537878513336182, -0.5288718342781067, 0.005627698265016079, -1.8040329217910767, 0.30971580743789673, -2.051912546157837, -2.352938413619995, 1.04588782787323, 1.3716928958892822, -0.33353883028030396, -1.0573227405548096, 0.06072958931326866, 1.63845694065094, 1.6448980569839478, 0.9861883521080017, -0.5568613409996033, -0.7504580020904541, -1.8983010053634644, 0.6122309565544128, -1.141297459602356, -0.7821332812309265, -0.568101704120636, -0.60783851146698, 0.8408923149108887, 0.14090892672538757, 1.0906648635864258, -2.07309889793396, -0.18754835426807404, -0.5007574558258057, 1.7713702917099, 0.018378302454948425, -1.786454677581787, -0.5763893127441406, -1.1901289224624634, -3.0419840812683105, -0.4060288965702057, 0.28219619393348694, 0.05827239155769348, -1.1959750652313232, -0.30107125639915466, -0.8094210028648376, 0.5761364698410034, 0.5350900888442993, -0.5880591869354248, -0.6547803282737732], [-0.27001240849494934, -0.6123524904251099, -0.4400007128715515, 0.030091971158981323, 0.3342815637588501, 0.3133041262626648, -1.3661717176437378, -2.4352335929870605, -0.1983056366443634, 0.5423302054405212, -0.9485028982162476, 0.428287535905838, -0.5892784595489502, -0.3049778938293457, 1.2619320154190063, 0.1977686733007431, -1.2348347902297974, -0.03254075348377228, -0.129373699426651, 0.5504558682441711, -1.4687211513519287, 0.34910809993743896, 1.0854541063308716, 0.4934636354446411, -2.2071897983551025, -0.09953924268484116, 0.09978365153074265, 0.1515493392944336, 0.07831532508134842, -0.733840823173523, -0.7394230365753174, -0.8347963690757751, -0.8211361169815063, 1.4295295476913452, -0.21366366744041443, 0.32476499676704407, -0.2222779095172882, 0.2693386971950531, -0.31022512912750244, 0.0318366140127182, -1.1830730438232422, -0.05922070890665054, -0.0038175855297595263, 0.3027643859386444, -0.1140102818608284, 1.2330249547958374, 0.10261543095111847, -0.28931981325149536, 0.14705006778240204, 0.9693486094474792, -0.6304630637168884, -0.8984050750732422, 1.3046036958694458, -0.5651467442512512, 0.16655388474464417, -0.03713417053222656, -0.9011557102203369, -0.1925336718559265, -1.4809552431106567, -0.9908183813095093, 1.0515509843826294, 0.08738236874341965, -0.3749626874923706, -1.4999059438705444], [-0.047317735850811005, -0.09385760873556137, 1.0651179552078247, 0.36444783210754395, -0.10694796591997147, -0.19184111058712006, -0.1623142808675766, -1.027431607246399, -0.7443885803222656, -0.5348619222640991, 0.6969444751739502, -0.06041247397661209, -0.4201899468898773, -1.1556639671325684, 1.260530948638916, 0.7486221790313721, -0.5799688100814819, 1.5252439975738525, 0.44673997163772583, 0.1329176127910614, 1.9347745180130005, 1.0689401626586914, 0.08898718655109406, -0.5038489103317261, -0.9246032238006592, -1.0823348760604858, -1.1094136238098145, -1.009480357170105, -0.7941739559173584, -2.2739970684051514, -1.7271485328674316, 0.7367054224014282, 1.0218576192855835, 1.9309921264648438, 0.8800797462463379, -0.7119352221488953, 0.33290785551071167, -0.25533750653266907, 0.20094774663448334, 0.5117154717445374, 0.8598297834396362, 1.343846082687378, -1.1906007528305054, 1.17013418674469, 0.3297804594039917, 0.1478520631790161, -0.20876775681972504, -0.4754529595375061, -0.3034035861492157, -1.78990638256073, 1.5742696523666382, -0.0025008232332766056, -1.0292458534240723, -0.21507519483566284, -0.6489223837852478, -0.811012327671051, -2.46134090423584, -1.2385185956954956, -0.39204367995262146, -0.8803041577339172, -0.6662536859512329, 0.4063514471054077, -0.40742576122283936, 0.2556299865245819], [-0.736612856388092, 0.14265897870063782, 0.4047677516937256, -0.1057620570063591, 1.508735179901123, 1.8960727453231812, -0.11744258552789688, -0.4152587056159973, -1.2248023748397827, 1.156110167503357, 0.7903843522071838, 0.9913739562034607, 0.4233420789241791, -0.17667673528194427, -0.7067493796348572, -0.10891131311655045, -0.6433545351028442, -1.2922286987304688, -0.7271872162818909, -0.6232137084007263, -0.5829896330833435, -0.6020302176475525, -0.7171822786331177, -1.20756196975708, 1.1837279796600342, 1.1720478534698486, -0.500012993812561, -1.6771506071090698, 0.14465953409671783, 0.42683205008506775, 0.7429432272911072, -0.40191230177879333, -0.16164807975292206, -0.2713114023208618, 0.14883172512054443, -1.434566855430603, 1.223217248916626, -0.6783334016799927, -1.0523035526275635, -0.06878907978534698, -0.9160003662109375, -1.1810699701309204, -0.754449188709259, -0.16365697979927063, -1.299637794494629, -0.13927093148231506, -1.5395913124084473, -1.157824993133545, 2.2624547481536865, 1.2462915182113647, -0.1561245322227478, -1.3917351961135864, 0.6670471429824829, -0.06569286435842514, 0.7173681855201721, -0.4299588203430176, -0.6615203619003296, 1.120487093925476, 0.3187884986400604, 0.6667423248291016, 0.3296428322792053, -0.9376842379570007, 1.1563467979431152, -1.1735830307006836], [1.9001258611679077, 0.553058922290802, 0.9078178405761719, -0.9057123064994812, 0.07688364386558533, -1.868527889251709, -0.4539307951927185, 0.22930729389190674, -1.7075755596160889, 0.982660174369812, 2.048612594604492, -1.4942666292190552, 1.9196659326553345, 0.09769070893526077, 0.3439735472202301, -1.8067492246627808, -0.1069445013999939, 0.9411748051643372, -0.8383992314338684, -0.45337894558906555, 0.697351336479187, 0.11240121722221375, 0.727441132068634, -1.0166674852371216, -1.5917059183120728, -0.561024010181427, -0.12699973583221436, -0.5900430679321289, -0.8557422161102295, 1.0262624025344849, 0.46628597378730774, -0.7093364000320435, 0.45694491267204285, -1.3551150560379028, 0.32451924681663513, 0.4710944592952728, 0.754253089427948, 0.8717301487922668, 0.7025929689407349, -0.12005846947431564, -0.1281711310148239, 1.585564374923706, -0.2405070811510086, -1.5603169202804565, -0.337467759847641, -0.23300287127494812, -0.26731806993484497, -0.29953712224960327, -0.16048289835453033, 0.35004377365112305, 0.5269346833229065, 0.0685102641582489, 1.6696776151657104, 0.5371522307395935, 0.5696247816085815, -0.032743215560913086, 0.7075609564781189, 1.252817988395691, 0.2272859811782837, 0.08095888048410416, 1.351157307624817, 0.6000678539276123, 0.397182434797287, 1.0808302164077759], [0.05324073135852814, 0.08012919872999191, -1.6557705402374268, -1.5992883443832397, 1.6301835775375366, 1.50979483127594, 1.0378361940383911, -1.1715089082717896, 0.16969148814678192, -0.3747847080230713, -0.4555637836456299, 0.1276850551366806, -1.1703251600265503, -3.497288465499878, 0.5418658256530762, -0.28692400455474854, 0.7845940589904785, 1.2082107067108154, -0.4198821187019348, -0.050581369549036026, -0.3745846450328827, -1.5153417587280273, 1.583068609237671, 1.5846091508865356, 0.9078906774520874, 1.8768149614334106, -1.1083556413650513, 0.33157244324684143, -1.0364508628845215, 0.7187159657478333, -0.6660274863243103, -1.2855974435806274, 1.9563872814178467, 0.14562898874282837, -1.0921030044555664, -0.8688298463821411, -0.09559488296508789, 0.3781387209892273, 1.648209571838379, 0.5475934743881226, -0.477214515209198, -0.5784468054771423, -0.7782356142997742, 0.4361308515071869, -0.6193701028823853, 0.6202443838119507, -0.22777202725410461, 0.851281464099884, 1.2266032695770264, -0.04316592589020729, -1.7881273031234741, 0.4682285487651825, 0.7931133508682251, 2.369384527206421, 1.57846200466156, -1.4527065753936768, 0.2668619751930237, -0.11856023967266083, 0.04684249311685562, -0.1817137449979782, -0.6991560459136963, -0.7933973670005798, 2.66520357131958, -1.4405410289764404], [0.7138299942016602, 1.466037631034851, -0.6227820515632629, -0.5190514922142029, 0.5090048909187317, -1.2678825855255127, 1.4301297664642334, 0.2149886190891266, -1.1478161811828613, 0.5497880578041077, 1.501389503479004, -0.025841763243079185, -2.0239808559417725, -1.3155038356781006, 0.4353845417499542, -3.6842219829559326, -0.08314505219459534, -0.7670077085494995, -0.9512300491333008, 0.5418640375137329, -0.46320873498916626, 1.0800074338912964, -2.270488739013672, 1.4756640195846558, -0.9475092887878418, -0.8159960508346558, 1.3797094821929932, 0.45418810844421387, 0.7100456953048706, 0.06492788344621658, -1.5790330171585083, -0.22973428666591644, 0.9930405020713806, 1.4430323839187622, -1.3720797300338745, -0.22316569089889526, 0.057012926787137985, -0.6291921734809875, -1.9905667304992676, -0.33134356141090393, 1.447483777999878, 0.4575193524360657, 0.7755222320556641, 0.6396995186805725, -1.935667872428894, -0.17733918130397797, 1.4994934797286987, -0.41650381684303284, -0.9854182600975037, 0.5212557911872864, -0.6835463047027588, -0.08675920963287354, 2.7295405864715576, -0.5947063565254211, 0.12571647763252258, 1.2023779153823853, -1.2019163370132446, 0.7356789112091064, -0.11566724628210068, 1.2181159257888794, 0.18478652834892273, -0.586976170539856, -0.9338014721870422, -1.1913989782333374], [-0.5138975977897644, -0.19938121736049652, -1.7684823274612427, 0.001256799092516303, -1.233185052871704, -0.9549800753593445, 1.0633949041366577, 0.2580248713493347, -0.07686039805412292, 0.3328671455383301, -1.105035662651062, -1.882328987121582, 1.2782899141311646, -0.05848728120326996, -0.8426694273948669, -0.6507498621940613, 0.5934873819351196, 1.4537341594696045, 0.752948522567749, -0.04993928223848343, -0.4927031099796295, 0.011325397528707981, 0.0010300436988472939, -1.2015234231948853, 0.7969159483909607, -0.35928839445114136, 0.20252856612205505, 0.30595171451568604, -0.574683427810669, 0.07015495747327805, 0.6513135433197021, 0.18742486834526062, -1.9326051473617554, 0.9398136734962463, -0.8286898732185364, -0.27809464931488037, 0.6803759932518005, 0.09500176459550858, -0.24755804240703583, -0.9794195890426636, -0.8998303413391113, -1.4035476446151733, -0.2920001447200775, -0.5611361265182495, -0.5637276768684387, 1.1535799503326416, 0.5053420066833496, -2.1679773330688477, -2.2883248329162598, -0.7598435282707214, 0.49055254459381104, 0.3876061737537384, 0.03931413218379021, -0.07094763219356537, -1.0173741579055786, -0.6628522872924805, 1.0089284181594849, -0.03945954889059067, -2.0344088077545166, 2.123408794403076, 0.6119073033332825, -0.8483802080154419, 0.10221754759550095, -1.523751974105835], [1.0651246309280396, 0.7741842865943909, -1.469693660736084, 1.1267284154891968, 0.08222601562738419, 0.9443867206573486, 1.144773006439209, -2.3130569458007812, 0.093715138733387, -0.21701660752296448, 0.8649176359176636, 1.4560465812683105, 1.45107102394104, -1.2672300338745117, 0.5051590800285339, -1.6344656944274902, -0.37742456793785095, 0.5536090731620789, -1.1198673248291016, 0.6821855902671814, -1.165891408920288, 1.3827224969863892, -0.10947422683238983, -0.625530481338501, 0.1245158463716507, 1.4893289804458618, -1.3875739574432373, -0.17369534075260162, 1.0719059705734253, 0.3505575656890869, -0.08689797669649124, -0.9288894534111023, 0.11220703274011612, -0.4479347765445709, -1.8703768253326416, -1.5231701135635376, 1.2099441289901733, 1.5335279703140259, 0.45603740215301514, -0.4726801812648773, -2.014324188232422, 0.5783113241195679, 0.36647284030914307, -1.7855634689331055, -0.6153693199157715, -1.0470762252807617, 1.5260800123214722, 0.2004937082529068, 0.18645446002483368, 0.17826224863529205, -1.3954493999481201, 1.9374096393585205, 1.9321831464767456, -0.6969040036201477, -0.16448694467544556, -0.4124482572078705, -1.4506062269210815, -0.7411272525787354, -0.5826477408409119, -3.000620126724243, -1.9068331718444824, 0.41939473152160645, 0.9194327592849731, -0.31234923005104065], [0.061284538358449936, -0.648036539554596, 1.0493841171264648, 0.422133207321167, 0.6242696046829224, -0.3979038596153259, -0.15585659444332123, -0.6878424882888794, 0.054288607090711594, 0.7324606776237488, 0.9666450023651123, 0.07132165133953094, 0.5593698620796204, 0.08907340466976166, -1.7833372354507446, -0.1900508552789688, -0.1980873942375183, 0.861081063747406, -0.797872006893158, 0.5526788234710693, 0.48813891410827637, -0.6797777414321899, -0.2350092977285385, -0.018634282052516937, 0.06344107538461685, -0.25030988454818726, 0.9114416241645813, -0.7529801726341248, 1.0392696857452393, 0.9987020492553711, 1.1039797067642212, 0.7223566174507141, -0.5420218110084534, 2.5526225566864014, -0.27335524559020996, -0.5562665462493896, -0.8511803150177002, -0.7617029547691345, -0.9557011127471924, 0.327970951795578, 1.8968976736068726, 1.1818853616714478, 0.174744114279747, 0.025875816121697426, 1.1291630268096924, -0.49382951855659485, -1.282120943069458, 0.670967698097229, 0.21682162582874298, 0.1987384706735611, 0.14638297259807587, -0.4690658748149872, -1.0120590925216675, 1.8896052837371826, 1.1771401166915894, -1.666507601737976, -0.40716248750686646, 2.2107622623443604, -0.14225918054580688, -0.43819883465766907, 1.2030574083328247, 0.5678467154502869, -1.1695142984390259, 0.29318246245384216], [0.9380958676338196, 0.942669153213501, -1.3156583309173584, 0.6267300248146057, 0.2956499755382538, 0.1924843192100525, -1.3594516515731812, 0.9313012957572937, -0.5501425266265869, -0.007043161895126104, 0.6182118058204651, 0.54491126537323, 0.4663018584251404, -2.0031399726867676, -0.6124897599220276, 0.9124671816825867, -0.5951145887374878, -0.5965256690979004, 0.8806061148643494, 0.16195017099380493, -1.1636592149734497, -2.2093536853790283, -1.661035180091858, 1.9858484268188477, 0.4237641990184784, -0.14419487118721008, 0.2869206964969635, 0.10767540335655212, -0.7469742298126221, -0.19546066224575043, 0.039556555449962616, -1.0120190382003784, -0.46166545152664185, -2.596620798110962, -0.9328013062477112, 0.8614767789840698, 0.007404425647109747, -0.9640252590179443, 1.289557933807373, -0.5403362512588501, 1.2067686319351196, 0.27963218092918396, 0.2607230544090271, 2.833667039871216, 1.0041924715042114, -1.655813217163086, 0.2542720437049866, 0.53724205493927, -0.06476103514432907, 2.1992433071136475, -1.3767879009246826, 1.6953327655792236, 0.6515044569969177, -2.8367767333984375, -1.3734744787216187, 0.31352803111076355, -0.9977180361747742, -0.5137259364128113, -0.9611904621124268, 1.5182961225509644, 1.0097031593322754, -0.1292174756526947, -1.1066474914550781, -2.6220908164978027], [-0.8343110680580139, -0.5880900025367737, 3.3216323852539062, 0.05598872900009155, 0.46922582387924194, 1.9259406328201294, 0.6708154678344727, -0.7361935377120972, 0.7958923578262329, -0.5472733378410339, 0.5013539791107178, 0.6275253891944885, -0.8984417915344238, -0.7309089303016663, -0.5488619208335876, 1.5030275583267212, -1.0416992902755737, 0.1299278885126114, -0.5970064997673035, 0.23717868328094482, -0.025418832898139954, -1.819019079208374, -0.6011616587638855, -0.4188811480998993, 0.12414372712373734, 0.3727427124977112, 2.666588306427002, -0.4710007905960083, 0.36373773217201233, 0.31587299704551697, 0.8374161720275879, 0.05586293712258339, 0.7832248210906982, 0.20269010961055756, -0.9639076590538025, 0.7476479411125183, 0.1881255954504013, 0.40301215648651123, -1.373558521270752, 0.05894859507679939, 0.23857536911964417, -1.5638231039047241, -2.9915337562561035, 0.41474759578704834, -0.5189134478569031, 0.3234935998916626, 0.2553163170814514, -1.6759345531463623, 0.34806501865386963, 0.7538738250732422, -1.0849026441574097, -1.2925786972045898, 0.3961181044578552, 0.1765117049217224, 1.1387474536895752, 0.8709263205528259, 0.013601923361420631, 0.2989819347858429, 0.728142261505127, -1.064186692237854, 0.976352870464325, 1.2632910013198853, -0.49678125977516174, -0.670735239982605], [0.21498973667621613, 1.7513229846954346, 0.18465039134025574, -1.4787914752960205, 1.1221705675125122, -2.2189512252807617, -1.329404354095459, -0.9911757111549377, -1.1158732175827026, -2.039947032928467, 0.5943924784660339, 0.14774975180625916, -0.38148120045661926, -0.6307933330535889, 0.036127857863903046, -0.07938627898693085, -0.03895900771021843, 0.5923646688461304, -0.8355963826179504, 0.5179834365844727, -0.3596832752227783, 1.392712116241455, 1.1679266691207886, 0.07519843429327011, -1.1955833435058594, 0.5419322848320007, -1.5951536893844604, 1.517074465751648, -0.26428791880607605, 1.886955738067627, 1.5191885232925415, 0.7493522763252258, -0.5819562673568726, 0.7115885019302368, 0.07920869439840317, -1.7219536304473877, -0.7966745495796204, 2.412808418273926, -1.524277687072754, 0.9201536178588867, -1.6703214645385742, 0.029076412320137024, -0.6119809746742249, -0.3897787034511566, 0.027273666113615036, -0.26761114597320557, -0.4964292347431183, 0.4525299668312073, 0.6442883014678955, 0.9926032423973083, -0.08209171891212463, 1.1064256429672241, 1.0708261728286743, 1.3569124937057495, 2.071776866912842, 0.3120601773262024, -1.9187346696853638, -0.2239251583814621, 1.3689686059951782, 0.08148261904716492, -0.3214688003063202, -1.536417841911316, 0.15274834632873535, 2.0393872261047363], [-0.5907621383666992, 1.6797404289245605, -0.3841782212257385, 0.7629527449607849, -1.1987534761428833, -1.0960630178451538, -0.15541569888591766, -0.6685957312583923, 1.5135952234268188, -0.6625322103500366, -1.0429062843322754, -2.413944721221924, 1.7119998931884766, -0.18764297664165497, 1.1684012413024902, 0.08854009956121445, -0.21710900962352753, 0.39208540320396423, 0.3881087005138397, -1.4352267980575562, 0.7318169474601746, -1.2252596616744995, 0.8394243121147156, -0.03935978561639786, 1.0198513269424438, -0.07803095132112503, -0.5116552710533142, -2.29689359664917, -0.5746099948883057, 0.8192169070243835, 0.5131176114082336, -0.7117551565170288, -0.7099539041519165, -2.4120705127716064, 0.7537910342216492, 0.19852717220783234, 0.1315869688987732, 3.089956045150757, 1.6032758951187134, 1.785652995109558, 0.4663700461387634, 0.4493664503097534, 1.2688188552856445, 1.5264052152633667, 1.3988784551620483, 1.9019458293914795, -0.10014814138412476, 2.2640304565429688, 0.5227685570716858, 1.0570387840270996, 1.3014720678329468, 1.1210798025131226, 0.21946029365062714, -0.10128484666347504, 1.1268423795700073, -1.2520586252212524, 1.5007572174072266, 1.730148196220398, 1.6696034669876099, -1.4156925678253174, -0.8267698287963867, 0.8666746020317078, -0.32280367612838745, -0.5707560777664185], [-0.24176034331321716, -1.6752979755401611, -1.3358579874038696, -1.1273130178451538, 0.21618393063545227, 0.2886897027492523, -1.2230392694473267, -2.07619047164917, -0.5718088150024414, 0.2753306031227112, 1.798323631286621, -0.8960900902748108, -0.2606976628303528, -1.1914565563201904, -0.08681152015924454, -0.1997992992401123, -0.12399697303771973, -0.46924710273742676, 0.5699758529663086, 1.0178554058074951, 1.4781572818756104, -0.20872044563293457, -1.1426719427108765, -0.7273082733154297, -0.6921478509902954, 1.4117854833602905, 1.3385734558105469, -2.194831609725952, 0.11653576791286469, -1.1253101825714111, 0.12464060634374619, -0.09020708501338959, -1.5548359155654907, -1.6647089719772339, -0.9078280925750732, 0.026081886142492294, -2.695131301879883, -0.25098374485969543, 1.524588704109192, -2.229531764984131, 1.898932933807373, -0.4922645688056946, -0.4723154306411743, 0.6203954815864563, 1.7347861528396606, -1.0904136896133423, 0.6151447296142578, 0.1875392198562622, 1.1066694259643555, -0.20132869482040405, 0.5193188786506653, 1.30461847782135, -0.884198009967804, -1.2102748155593872, -0.26569873094558716, -0.25671517848968506, -0.23434875905513763, -0.9523740410804749, 0.3839641809463501, -1.095476508140564, 2.3952250480651855, 0.527180552482605, 0.1858225166797638, -2.3345465660095215], [1.3321006298065186, -0.3257758319377899, -0.513479471206665, -0.9735689163208008, -1.3361806869506836, -1.2744503021240234, -0.5791977643966675, -1.1069262027740479, 0.13554897904396057, 0.05286560207605362, -1.8785136938095093, -0.05064602941274643, 1.3767585754394531, 0.1885703057050705, -0.8316261172294617, 0.16184911131858826, 0.8719722032546997, 1.4489771127700806, -0.7631573677062988, 0.06244178116321564, 1.3192754983901978, -0.2573702037334442, -0.8622728586196899, -0.48009732365608215, -0.32347050309181213, 1.230922818183899, 1.1325966119766235, -1.0900970697402954, 0.058866798877716064, -0.1935129314661026, 0.8074177503585815, -0.696792483329773, -0.29089048504829407, -1.2629873752593994, 0.2950568199157715, 0.8821449875831604, 1.3462778329849243, 0.026668230071663857, -0.42339056730270386, -0.9685013890266418, -1.9767695665359497, 0.11742162704467773, -1.299829125404358, -0.15028131008148193, -0.5771588683128357, 1.5712416172027588, -1.0857120752334595, 2.000947952270508, -3.027776002883911, -1.2188817262649536, -1.2024935483932495, -1.2026002407073975, 0.7708503007888794, -0.11272088438272476, -1.5272102355957031, 0.4447948932647705, -0.8744262456893921, -0.028315505012869835, -0.5467594265937805, 0.28030747175216675, -1.1723179817199707, 1.4125661849975586, 0.26312145590782166, 1.9717185497283936], [-0.5910250544548035, 0.1339203268289566, 1.5151407718658447, -2.4689786434173584, 1.36812162399292, -1.0770025253295898, 0.48491257429122925, -0.47352197766304016, 0.6095877885818481, -1.4394086599349976, -0.1419302523136139, -3.1137707233428955, 0.8395179510116577, -2.46636962890625, -1.2151213884353638, -3.476655960083008, -0.5322948694229126, -1.1863300800323486, 0.5202842354774475, -1.2459582090377808, -1.6430330276489258, -0.3159693777561188, -1.3530926704406738, 0.14658167958259583, 0.9716461896896362, -0.4505736231803894, -1.050164818763733, -0.1148335263133049, -0.8760232925415039, -0.5957619547843933, 1.077960729598999, 0.3593250513076782, -0.42555058002471924, 0.270522803068161, -0.2564164698123932, -1.7431570291519165, 0.1931726336479187, -0.25805607438087463, -0.8405934572219849, -0.02595703676342964, 0.5601637363433838, -0.6087201237678528, -0.8028730154037476, -0.03457970917224884, -2.351592779159546, -0.8562254905700684, 0.07421764731407166, 0.37768247723579407, -1.3919463157653809, 1.449626088142395, 0.19815194606781006, 0.6171915531158447, 0.7593110203742981, 1.5889568328857422, -1.0829180479049683, 0.450370192527771, -2.1162233352661133, -2.232994318008423, -0.44754692912101746, -0.1394113153219223, 0.38487735390663147, 0.7999454736709595, 1.1133068799972534, -0.8583405613899231], [2.627534866333008, 0.5725499391555786, -0.4563243091106415, 0.02519235946238041, -0.7789056301116943, -2.6874377727508545, -0.9347429871559143, -0.6048067808151245, -0.11611475050449371, 0.33517274260520935, -0.39066845178604126, -0.6170730590820312, 0.530002236366272, -0.17886121571063995, -1.2604734897613525, 0.9986321926116943, 0.22739991545677185, -0.2533060908317566, 0.6753969788551331, 0.25365760922431946, -0.8633459210395813, -0.8734366297721863, 0.1373452991247177, -0.23418183624744415, 1.0217233896255493, 0.2093680053949356, -1.7563138008117676, -0.7204289436340332, -0.7440032958984375, 0.10321294516324997, -0.20320121943950653, 0.5403205156326294, 1.2130380868911743, 0.6381548643112183, 0.6863439679145813, -0.29141852259635925, -0.8565978407859802, -2.63454532623291, 0.592646062374115, 0.5999248623847961, 0.6303354501724243, -0.6653589010238647, -0.39817383885383606, -1.0119831562042236, 0.13022653758525848, 0.33539167046546936, 1.0734294652938843, 0.22595946490764618, -0.563795804977417, -1.018169641494751, 0.5436511635780334, -0.8871760964393616, 0.6370684504508972, 1.695070743560791, -1.0366991758346558, -0.5334669947624207, 0.24235652387142181, -0.21388638019561768, 1.3120061159133911, 0.8141505718231201, 0.4818880259990692, 0.21490994095802307, -1.247872233390808, 1.3829110860824585], [-1.32225501537323, 1.4519072771072388, 1.2121310234069824, 0.06198684498667717, -0.4074544906616211, -0.4789975881576538, -0.4754129946231842, -1.1534816026687622, 0.11357277631759644, -0.5467931032180786, 0.04770330712199211, 0.5537205338478088, 1.18617844581604, -0.3090782165527344, -0.014137296006083488, -1.8039647340774536, 1.207565188407898, 0.06285654753446579, 0.6646934747695923, 0.038660258054733276, 1.5445635318756104, -0.20391947031021118, -0.12365730851888657, -0.020253943279385567, -0.2449907660484314, -0.06352904438972473, 1.7819520235061646, 1.8651612997055054, 0.1536281853914261, -0.7136954069137573, 1.4189484119415283, -1.137621521949768, 0.1394198089838028, 1.5560778379440308, 0.46948033571243286, -0.8964629173278809, 0.1703016310930252, -0.6646537184715271, 0.21650078892707825, 1.859370470046997, -0.8979912996292114, 0.0716223418712616, 0.05546506121754646, -0.04158152639865875, 0.712482213973999, -0.1441485434770584, -0.2927037179470062, -1.0055986642837524, 0.28011950850486755, 0.9108927845954895, -0.5754347443580627, 0.05437089502811432, 0.0990363284945488, -0.3321865499019623, -1.5474897623062134, -0.11625069379806519, 0.7985695600509644, 1.245255708694458, 1.7772328853607178, -0.5345876812934875, 0.9526739716529846, 0.521302342414856, 0.713546633720398, 0.0036186762154102325], [-0.21211661398410797, 0.06916338950395584, -1.012592077255249, -1.2491636276245117, -1.3545442819595337, -1.4699623584747314, 0.28907889127731323, 0.7473191618919373, -0.5830340385437012, 0.7245861291885376, -1.5448472499847412, 0.9899083971977234, -0.25944775342941284, -0.3961227238178253, 0.5964257717132568, -0.28582897782325745, -1.4753203392028809, 0.9117556810379028, -1.3597602844238281, -1.872793436050415, -0.6490908265113831, -0.02216702699661255, 0.16605934500694275, 0.036584097892045975, 0.22127921879291534, 1.9300001859664917, -1.0864778757095337, -0.7255082726478577, 1.2646725177764893, -0.9017642140388489, 0.7376013994216919, 1.7585221529006958, 0.3323281705379486, -0.5493006110191345, -1.277551531791687, 2.56046462059021, -0.2096167653799057, -0.6403793692588806, -2.6191394329071045, -0.6748009920120239, -0.1569572240114212, 1.2515820264816284, 0.44225072860717773, 0.9655420780181885, 0.3099331259727478, -0.14608998596668243, 1.6390498876571655, -1.3769935369491577, 1.9498142004013062, 1.2349021434783936, 0.6392821073532104, -1.2378565073013306, 0.7234596610069275, 0.14936129748821259, 0.9731003642082214, -1.2732101678848267, 0.40522855520248413, -0.6170580983161926, -2.016977310180664, 0.2977403700351715, -0.3091013431549072, -0.6621031165122986, 0.12176672369241714, 0.1501619815826416], [1.6635873317718506, 1.1762969493865967, -1.05575692653656, -1.1838654279708862, 1.3740209341049194, -0.9218617677688599, 0.27515920996665955, -0.3247108459472656, 0.7898687124252319, -0.9652777910232544, 0.22400225698947906, 0.9269425272941589, -0.11527859419584274, -0.9342854619026184, 0.0027864803560078144, 0.539188027381897, 0.09850209206342697, -1.3674155473709106, 1.837180256843567, 0.44908973574638367, 0.4952358603477478, 1.5136741399765015, -0.9662674069404602, 0.3044271171092987, 1.9949204921722412, 1.0762903690338135, 0.9920650124549866, 1.542136549949646, 0.038466889411211014, -0.1548617035150528, 0.035707611590623856, -0.49791693687438965, 0.31227099895477295, 1.8951363563537598, -1.6817147731781006, 0.9550058245658875, 0.41887688636779785, 0.969516396522522, 0.9570636749267578, 0.6704278588294983, -1.7204105854034424, 0.587714433670044, 1.472141981124878, -0.24206580221652985, -0.7806786298751831, -0.9902268648147583, -0.35130637884140015, -0.2506907284259796, -0.9956263303756714, -0.12305480241775513, -1.5309096574783325, -1.2339632511138916, 1.3001450300216675, -0.20003892481327057, -0.5351309180259705, -0.7902110815048218, -0.14966411888599396, 0.8116168975830078, 0.3502955734729767, 1.5925770998001099, -0.24911057949066162, -0.8980110287666321, -0.49081650376319885, -1.250722050666809], [1.1013621091842651, -0.12435903400182724, -0.6283811926841736, -0.8881159424781799, 0.46938571333885193, 0.4548957347869873, 1.2534732818603516, -3.043185234069824, 0.7390781044960022, -0.40313056111335754, 1.1324563026428223, 1.0506469011306763, -0.1993219405412674, -0.4699536859989166, 0.33587750792503357, 2.9966306686401367, -1.5592155456542969, -0.9597996473312378, -0.44541630148887634, 0.16935686767101288, -0.13612264394760132, -0.4821925461292267, -0.6520529389381409, 0.86634361743927, -0.6016254425048828, -0.5864843726158142, -0.22448478639125824, -1.0340192317962646, 1.2655720710754395, -0.6570536494255066, -1.0577621459960938, 0.5768439173698425, -0.6246723532676697, 0.11027158796787262, -1.1296334266662598, -0.06993968039751053, -0.16626200079917908, 0.21734343469142914, 0.37389305233955383, -0.5850539803504944, -0.23357361555099487, 0.7710363268852234, 1.0775882005691528, 1.6473822593688965, -0.6336488723754883, 0.529691219329834, 0.3672809898853302, 0.2227909117937088, 0.7657807469367981, 1.376312255859375, -0.6284797191619873, -0.620898962020874, 0.714768648147583, 0.5344058275222778, -0.5917705297470093, 1.0817291736602783, -1.1435086727142334, -0.31575021147727966, -1.4360015392303467, -0.1632365882396698, -1.8873116970062256, -0.34735316038131714, -0.15575256943702698, 0.6506600379943848], [-1.175397276878357, -0.010530083440244198, 0.4108569920063019, 0.5938589572906494, 0.36787715554237366, 1.154680609703064, -0.00041502786916680634, -2.365171432495117, 0.5379239916801453, 0.965697169303894, 0.1930236965417862, 2.7301926612854004, 0.35395485162734985, -1.786588191986084, -0.4462125599384308, -0.839747965335846, 1.4153475761413574, -0.788194477558136, -1.7562710046768188, 0.32367226481437683, 1.5811797380447388, 2.6314404010772705, 0.6713865399360657, -0.2894288897514343, 0.5941401720046997, 0.8126402497291565, 1.6992261409759521, 0.01278079953044653, 0.008386692963540554, 0.8065884113311768, 0.4021381437778473, 0.9458526968955994, 0.23474714159965515, 3.357605457305908, 0.0037769984919577837, -0.6532675623893738, -0.8915419578552246, 1.7804661989212036, -1.2837404012680054, -0.6898491978645325, 0.06955279409885406, 1.2261022329330444, 1.241687536239624, 0.6848006844520569, -0.15646347403526306, -0.35765284299850464, 0.6798252463340759, -0.25602221488952637, 1.4314013719558716, 1.0050508975982666, 0.144028440117836, 1.323521614074707, -1.1051297187805176, -0.4191964566707611, -0.5045944452285767, 0.42118746042251587, -0.4967575669288635, 2.017092704772949, 0.4417227804660797, -0.583279013633728, -0.7227535843849182, 0.8711472153663635, -0.7094005346298218, -0.8028486967086792], [-0.15723201632499695, -1.2480188608169556, -1.2617371082305908, -1.6437721252441406, 0.6218613982200623, 1.184862494468689, -0.5442471504211426, -1.1862540245056152, 0.973771333694458, 2.31307315826416, 0.3806281089782715, -0.8304443359375, 0.04226420819759369, 1.5325926542282104, -0.4991127550601959, -1.923290491104126, 0.024982623755931854, -0.12677234411239624, 1.1464731693267822, 0.25740480422973633, 0.3944462537765503, 1.1679071187973022, 0.6703753471374512, 0.10095876455307007, -0.7778853178024292, 0.7496238350868225, -1.7668719291687012, -0.38980013132095337, 1.0390007495880127, 0.7535949945449829, 1.5832606554031372, -1.4035372734069824, 1.3300588130950928, 0.029719296842813492, -0.5180780291557312, 0.8762938976287842, 0.4226692318916321, -0.23700742423534393, -1.005013108253479, 1.6190708875656128, -1.6836826801300049, -0.9465206861495972, -1.2648260593414307, 0.9595857858657837, -0.5232081413269043, -0.952919065952301, -0.7698501348495483, 0.29656702280044556, 0.7261009812355042, -1.4305764436721802, 0.6097083687782288, -0.19758863747119904, -0.4965001046657562, 0.17637698352336884, -0.22233162820339203, -0.27744728326797485, -0.9232456684112549, -0.7111122608184814, 1.1988871097564697, -0.7537535429000854, 0.20026405155658722, 1.4480156898498535, 0.554556131362915, 0.9674885869026184], [-0.007352180313318968, 0.4290527105331421, -2.2396578788757324, -0.33899930119514465, 0.1702546328306198, 1.4887367486953735, 2.1368255615234375, -1.5792664289474487, 1.9643720388412476, -1.6009998321533203, -0.7385040521621704, 0.7689646482467651, 0.29877281188964844, 0.005259535275399685, 0.5334526300430298, -0.28500881791114807, -1.5634629726409912, -3.4800703525543213, 0.4222472012042999, -0.07809367030858994, 1.3623250722885132, 0.9373536109924316, 0.7409696578979492, 1.5060337781906128, 0.05992351472377777, -0.5073205232620239, -0.19392046332359314, 1.9888417720794678, 0.2557624280452728, -1.5955885648727417, -0.6939773559570312, -0.004743475932627916, 0.213237002491951, -0.5076355934143066, 0.09690514951944351, 0.5794228315353394, 0.6827378273010254, 0.3817821741104126, 0.31279459595680237, -0.47406911849975586, 0.15287570655345917, 1.9472439289093018, -0.6887208819389343, -0.07502583414316177, 0.15884427726268768, 1.5259077548980713, 0.38210535049438477, 1.3916451930999756, -0.2244284749031067, 1.578000545501709, 0.3253459632396698, -0.6222715377807617, 0.11026154458522797, -0.351529598236084, -1.2095369100570679, -0.6063295006752014, -1.0139250755310059, 2.0126724243164062, -0.11647259443998337, -0.2947081923484802, -0.8478469252586365, -2.1408684253692627, 1.3382610082626343, -0.015450994484126568], [-0.3717118799686432, 0.8957429528236389, -0.2982650101184845, 0.33237171173095703, -0.11691704392433167, 0.4400060176849365, 0.2556535005569458, 0.5223926901817322, 0.8522711396217346, 0.1321294605731964, -1.5770918130874634, 1.3575856685638428, -0.15299443900585175, 1.904751181602478, -1.1949766874313354, -1.9488229751586914, -1.9977587461471558, -1.5337917804718018, -0.703140139579773, 0.5824487805366516, -1.6681932210922241, 1.6000659465789795, 2.1003050804138184, -0.43502670526504517, 1.4837692975997925, 1.1301894187927246, -0.8793239593505859, 1.6430472135543823, -0.31876564025878906, 0.925674319267273, -0.016310280188918114, -0.1833653301000595, 1.4833950996398926, -1.0439209938049316, -0.5164503455162048, -0.8769104480743408, 0.9498944282531738, 1.0650794506072998, 0.4855683147907257, -0.38449838757514954, 0.9569316506385803, 0.5566492676734924, -1.052185297012329, -0.7250556945800781, 0.5641300678253174, -1.527241587638855, 1.257682204246521, 0.8920554518699646, 0.30360960960388184, -2.0063722133636475, 0.08325366675853729, -0.30998459458351135, -0.15148767828941345, -0.5739396214485168, -0.210246741771698, 0.6309072971343994, 0.1077006533741951, 0.7503290772438049, -0.027905065566301346, 2.4172511100769043, -0.6205117702484131, -0.031834084540605545, -0.540522038936615, 0.7042736411094666], [-0.902386486530304, 0.3402498960494995, -1.1132256984710693, 0.6966904997825623, -0.9975363612174988, 0.3338431417942047, 0.9176588654518127, 0.3308333456516266, -0.6756972670555115, -1.9823046922683716, 0.3953182101249695, -0.18611328303813934, 1.193718433380127, 0.48463454842567444, -1.3775733709335327, 0.5078416466712952, 2.0786032676696777, -0.7696993350982666, 0.7525781989097595, 0.6107014417648315, -0.022237803786993027, -1.6303348541259766, 0.8948233723640442, 1.2623437643051147, 0.28171131014823914, -1.5604701042175293, 0.049876876175403595, 1.5243788957595825, -1.0694795846939087, -0.5060457587242126, 0.05622413009405136, -1.6598857641220093, -0.04527841880917549, 0.579149603843689, -1.007445216178894, 0.5872838497161865, -0.3016601800918579, 0.2004082202911377, 0.20451462268829346, 0.8852524161338806, 1.8632030487060547, 0.8104101419448853, 0.37294915318489075, -0.9928184747695923, -0.03292674571275711, 0.0434381365776062, -1.8995587825775146, -1.6496583223342896, 0.5974043011665344, 0.341749370098114, 0.18655435740947723, -0.8986107110977173, 1.8329272270202637, -0.7886954545974731, -0.8459658622741699, 0.7785531878471375, -1.0724457502365112, -0.7319452166557312, 0.21083460748195648, 0.9655543565750122, -0.8402061462402344, 0.07972890883684158, 0.1432245671749115, 0.6255348324775696], [-0.7952348589897156, -0.587915301322937, -0.37880977988243103, 0.882585883140564, -0.13022403419017792, 0.46514812111854553, -1.0097200870513916, 0.8115600943565369, -0.09476840496063232, 0.5679195523262024, 0.02313900552690029, -0.1408127099275589, 0.562303364276886, 0.6059932112693787, 0.02257848158478737, 2.2965197563171387, 0.9174572825431824, 2.479139804840088, 1.073974609375, -0.7149494290351868, -0.06954654306173325, 1.001280665397644, 0.6776823997497559, -2.8425090312957764, -0.33198487758636475, 1.2229739427566528, 1.0112059116363525, 0.8949400186538696, -1.6586593389511108, -0.2645120620727539, 1.019673466682434, 0.021128932014107704, 1.3087117671966553, -1.0149093866348267, -1.2429790496826172, 1.3114980459213257, 0.07881781458854675, 1.4543728828430176, 0.6414923667907715, 1.642441749572754, -0.34584805369377136, -0.12830443680286407, 1.5584579706192017, 1.0010677576065063, -0.48033592104911804, 0.8607219457626343, -2.7815427780151367, -0.14934512972831726, 0.09822610020637512, -1.1217926740646362, 0.85912024974823, -0.182032972574234, 2.9453227519989014, -1.6973044872283936, -0.3692944347858429, 0.8863190412521362, 1.3868916034698486, -0.3560200333595276, 0.3862529993057251, 0.38998687267303467, 0.04513586685061455, 1.4029293060302734, -0.14086230099201202, 0.6132224798202515], [-0.5259303450584412, -0.9814490079879761, -0.9725753664970398, 2.231558084487915, 0.3532746434211731, 0.3195940852165222, 0.34391021728515625, -0.2720296382904053, 0.8361462354660034, 0.7284108996391296, 1.210479736328125, -0.7527768015861511, 0.9368517398834229, 0.7294082045555115, 1.615541696548462, 0.21827799081802368, -0.42308929562568665, -0.9121231436729431, 0.16793733835220337, -0.4913787543773651, -0.7447412014007568, -0.9936920404434204, 0.48043203353881836, 0.49894529581069946, -1.2263754606246948, -0.5866307020187378, -1.573742151260376, -0.6751084327697754, -0.1766113042831421, -1.3880587816238403, 1.409331202507019, -0.35445350408554077, -0.3465780019760132, -2.275604486465454, -1.4629757404327393, -0.3518138825893402, 1.0229721069335938, -0.37186866998672485, -1.4552110433578491, 0.8922025561332703, -1.64418363571167, -0.17028990387916565, 0.07075940072536469, -2.4362053871154785, -3.137336254119873, -1.0619693994522095, -0.43958204984664917, -0.17113880813121796, -0.8316803574562073, -0.8453114628791809, -0.7352427244186401, -0.21037615835666656, -2.5250847339630127, 0.708047091960907, 0.127309650182724, 0.314626544713974, 1.1037859916687012, -0.41450220346450806, 0.8955549001693726, 0.9313945770263672, 0.2607327997684479, 2.4717085361480713, -0.08339136838912964, 0.1528138965368271], [0.3877139091491699, 0.6481043100357056, 0.3096441328525543, -1.4465364217758179, -2.10183048248291, -0.5822603106498718, 1.7354927062988281, -0.3251413106918335, 0.9447188377380371, 0.12934860587120056, -0.7901245355606079, -1.1761679649353027, 1.7505252361297607, -0.5228368639945984, -0.343696653842926, -0.4811340570449829, 0.4858129620552063, 1.2443463802337646, 0.03106403723359108, 0.011056376621127129, -2.2157394886016846, 2.0780630111694336, -0.8185314536094666, -0.19126646220684052, 0.786871612071991, 0.6600562930107117, -0.6800233125686646, -2.719186305999756, 0.4937579929828644, 0.05832419544458389, 0.5511981248855591, -0.4476132392883301, -0.1208086684346199, 0.03979254886507988, 1.9565885066986084, -0.28792625665664673, 0.9471656680107117, -0.4026002287864685, -1.2829194068908691, -1.6250935792922974, 1.2706965208053589, 0.9147748351097107, -0.10392512381076813, 0.6051209568977356, -0.5150982141494751, -2.046842098236084, 0.3083065450191498, 0.19804596900939941, 0.19637064635753632, -1.059938907623291, 1.3833611011505127, -0.587765634059906, -0.8079587817192078, 0.38132333755493164, -0.7496762275695801, -0.3264065980911255, -1.8900401592254639, 1.9811363220214844, -0.6225754618644714, -0.3011384904384613, -1.0315446853637695, 0.42656832933425903, -0.5107864141464233, 0.11991344392299652], [1.0429273843765259, 0.25085875391960144, -0.5276687741279602, -0.48864930868148804, 1.355480432510376, 0.37370485067367554, 0.7549808621406555, 1.1157689094543457, 1.4350961446762085, -1.7243468761444092, -0.5196657776832581, 1.99321711063385, 1.4821650981903076, 0.5937749743461609, 0.35168105363845825, 0.21799419820308685, -0.40113043785095215, -0.6502838134765625, 0.47948628664016724, -0.14348635077476501, -0.38737791776657104, 0.4295041263103485, 0.0884503722190857, -0.6242157816886902, 0.674640417098999, 0.8953620195388794, 1.1600565910339355, 1.0520490407943726, 0.13611792027950287, 0.5018543004989624, 0.03754302114248276, 0.9884432554244995, -0.2916557788848877, -0.9405162334442139, -0.9517700672149658, -0.36571627855300903, -0.9306395649909973, -0.1279933750629425, -0.31409820914268494, -0.9412750005722046, -0.1786489486694336, 0.30959999561309814, -1.3554364442825317, -0.392476350069046, -1.5385410785675049, 3.1020026206970215, -1.657788634300232, -0.8675140142440796, 0.63978111743927, -1.069100260734558, 2.1453356742858887, -0.10621009021997452, 0.6809348464012146, -1.047504186630249, -0.7718592882156372, 1.4176743030548096, 1.7525893449783325, -0.060029324144124985, 1.9562551975250244, 0.03017844259738922, -0.9945406913757324, 0.05513422191143036, 0.48625221848487854, -0.7030372023582458], [-1.9357378482818604, 0.6559765934944153, 1.260619044303894, -1.6038434505462646, 0.5853047370910645, -0.5252652764320374, -0.007052091881632805, 1.0441858768463135, 2.780635356903076, -2.013875722885132, 0.6461285352706909, -0.4590076208114624, -0.8433110117912292, -0.20094017684459686, -1.611544132232666, -1.1157978773117065, 0.3676545023918152, -1.4723083972930908, -1.1599587202072144, 1.1682919263839722, 0.028721848502755165, -0.7358217835426331, 0.11578971147537231, -1.4666441679000854, -0.7687904834747314, -0.7269445657730103, 0.5739174485206604, -0.10526416450738907, 0.8423616886138916, 0.5412598252296448, -0.231942817568779, 0.006457394454628229, -0.29476919770240784, 0.3493894338607788, -1.433868646621704, 0.3586360812187195, 1.0269644260406494, -0.4060947299003601, 0.34914737939834595, -0.20310822129249573, -0.3262394368648529, -0.9864195585250854, -0.4695972502231598, -1.8151934146881104, 2.81693696975708, 1.433685064315796, 1.7018340826034546, -1.1015034914016724, -2.669886827468872, 0.8675317764282227, -0.15564434230327606, -1.1491036415100098, -0.05502281337976456, 0.048555657267570496, 0.3384324610233307, -1.2859424352645874, 1.796481966972351, 1.0439766645431519, 0.053410712629556656, -2.7056937217712402, -1.2362141609191895, 0.46694254875183105, -1.4309637546539307, -1.7893991470336914], [-1.5479615926742554, 0.6482291221618652, -0.2028326839208603, -1.7911391258239746, -0.8708634972572327, 1.8054172992706299, 0.17589464783668518, 1.2713263034820557, -0.3365994989871979, 0.9292430281639099, 0.036063503473997116, 1.3717278242111206, -0.23520232737064362, 1.2752587795257568, -0.959905207157135, -0.28654393553733826, -0.9420338869094849, 1.346881628036499, -0.0408194400370121, 0.14208541810512543, 1.0084307193756104, 1.574561595916748, -1.107702374458313, -0.8738059997558594, -0.6650064587593079, -0.4767451286315918, -1.536444067955017, -0.19152846932411194, 2.3655622005462646, -0.9272844195365906, 0.5476590991020203, -1.0147956609725952, 0.586556613445282, 1.1472212076187134, -1.0159817934036255, 0.6322290897369385, -0.2951315641403198, 0.14792805910110474, 1.5078434944152832, -0.33999213576316833, -0.762390673160553, 0.6205984950065613, 1.0792454481124878, 0.39046865701675415, -0.8089315295219421, 1.3165671825408936, 0.7293813824653625, 0.00021021846623625606, 0.4946698844432831, -0.31169870495796204, 3.178107500076294, 0.9110758900642395, -1.7023049592971802, -0.8874608278274536, 0.8463340997695923, -0.35850244760513306, 0.8500787019729614, 0.08479200303554535, 0.1706477403640747, -0.10596544295549393, -0.536790668964386, 0.5188387036323547, -0.02250255085527897, -0.8086886405944824], [0.9663563966751099, -0.845749020576477, 0.43043947219848633, 0.08098221570253372, 0.09962679445743561, 0.3446511924266815, -0.04015624150633812, 0.7330719828605652, -1.6663002967834473, 0.9634831547737122, 2.749255418777466, -0.3051692247390747, -0.09704528003931046, -0.2579532563686371, -0.38448843359947205, -0.28130289912223816, 0.6957297325134277, 0.9895483255386353, 1.0952543020248413, -1.10369074344635, 0.6557153463363647, -0.289059042930603, 1.5107910633087158, -1.0584567785263062, -0.02682879939675331, -1.2752556800842285, -0.021919772028923035, 1.5708128213882446, 0.20026330649852753, 1.2328275442123413, -0.6151221394538879, -0.25426486134529114, 0.34788304567337036, -0.2031244933605194, 0.19623948633670807, -0.23093071579933167, 0.5720365643501282, -1.6847150325775146, -0.9949768781661987, -1.1027905941009521, 0.7020654678344727, -0.8417136669158936, -1.5901354551315308, -0.11149269342422485, 1.2632782459259033, 0.5062222480773926, -0.3427761197090149, -0.3891165852546692, -1.0402648448944092, -1.237374186515808, -0.4457891583442688, 0.7420948147773743, -0.24944545328617096, -0.4870460629463196, 0.40432071685791016, -1.4644607305526733, -1.2849056720733643, -0.3054860532283783, -0.18878258764743805, -0.9177943468093872, 0.4613724648952484, 0.28248342871665955, 0.13965915143489838, 1.4117482900619507], [1.0334126949310303, -0.9692527651786804, 1.2134511470794678, -1.3059123754501343, 2.1419403553009033, 0.8999355435371399, -1.037731647491455, 0.9944342970848083, 0.21362893283367157, -0.9570899605751038, 1.7828333377838135, 2.569030284881592, 0.7440705299377441, 1.2765814065933228, 0.27642685174942017, 0.26115506887435913, 0.5660004019737244, 0.1896587461233139, -2.835033893585205, 0.7953327298164368, 0.9639089107513428, -0.7521575689315796, -0.42999082803726196, -1.541177749633789, -0.6124933362007141, -1.61002516746521, -0.08704764395952225, -0.015483018942177296, 0.8052489161491394, -0.9399408102035522, 0.16923104226589203, -1.143530249595642, -0.3512009382247925, 0.07664963603019714, 0.6871938705444336, 0.5281939506530762, 0.03460986539721489, -0.05647704005241394, 0.40723323822021484, -2.0950262546539307, 0.24314124882221222, 0.5069475769996643, -0.37950584292411804, 1.6414908170700073, -0.8122727274894714, -1.2740730047225952, -0.34018468856811523, -2.458164691925049, -2.474426031112671, -1.431363582611084, 1.204345941543579, -0.05301178619265556, 1.0412826538085938, 0.5406750440597534, -0.41213780641555786, 1.577371597290039, 1.5179948806762695, 0.8120543956756592, 0.8521631956100464, 0.07216276973485947, -1.545810580253601, 0.8964914679527283, 1.3815325498580933, 0.2580408751964569], [-0.11017768830060959, -1.2070868015289307, -0.11226174980401993, 0.22398358583450317, -0.16265906393527985, 1.800751805305481, -1.3672553300857544, -0.3229849934577942, -0.5239607095718384, -0.5947790145874023, -0.49988478422164917, -0.706138014793396, -0.11704736948013306, -0.7601978182792664, 0.4945509135723114, 0.6494044661521912, -0.5251383781433105, -1.460677981376648, -2.223735809326172, 1.1298019886016846, -1.135696530342102, -0.8219608664512634, -1.107493281364441, 0.1298382431268692, 1.6113693714141846, -0.7559566497802734, -1.0802661180496216, -1.0378267765045166, -0.07277742773294449, -1.352644681930542, 2.834775447845459, 0.9726196527481079, -1.4721624851226807, -0.08680850267410278, -0.012944335117936134, 1.1797070503234863, 0.17558664083480835, 0.20990288257598877, 0.5288240313529968, 1.3291901350021362, -1.5771900415420532, -0.49027591943740845, -0.5792213082313538, -1.24655020236969, -0.356195330619812, 0.31042900681495667, 0.9671396017074585, 0.4169843792915344, 0.6527590751647949, 0.787555992603302, 0.39601385593414307, -1.1949188709259033, 0.8884963393211365, 0.31514185667037964, 2.1572418212890625, -0.32680606842041016, 1.4448332786560059, 1.0532567501068115, 1.3237146139144897, -1.0198419094085693, 0.8148450255393982, -0.8556793928146362, 0.009543764404952526, -0.5429645776748657], [1.1602855920791626, -1.4127241373062134, 0.26107916235923767, -0.19529522955417633, 0.517081081867218, 1.3466182947158813, -0.19772109389305115, -0.875849187374115, 1.8382525444030762, 0.5998947024345398, 0.12955063581466675, -1.3182663917541504, 0.6399862170219421, -0.42022550106048584, 1.5347259044647217, 0.8346057534217834, -1.8137024641036987, -0.37660256028175354, -0.004784342832863331, -0.32604414224624634, -1.1487482786178589, -1.464914321899414, 0.9061358571052551, -1.2368485927581787, 0.4638444781303406, 1.4599196910858154, 0.2091597467660904, 0.9715157747268677, -1.5752408504486084, 1.6827824115753174, -1.0760753154754639, 1.620113730430603, -0.2952820062637329, -0.4431755244731903, -1.4744340181350708, 0.08486603200435638, -1.7128123044967651, 0.48341867327690125, 0.7506300806999207, -1.8826026916503906, 0.2782917320728302, -1.0881301164627075, -0.9012476205825806, -0.5159333944320679, 1.039652943611145, 0.005321068689227104, -0.30350473523139954, 0.3279730975627899, -1.4903712272644043, -0.032497588545084, -1.2707030773162842, -1.441660761833191, -0.572993814945221, 0.46702247858047485, 0.14038130640983582, -0.9831104874610901, -1.1239758729934692, 0.0383155532181263, -2.1118059158325195, 1.77920401096344, 0.900239884853363, -0.8100539445877075, -0.46825888752937317, -0.6335318684577942], [0.14170248806476593, -0.6327770352363586, -0.3982267677783966, 0.1052083820104599, 0.4980895519256592, -2.1141116619110107, 1.2536702156066895, -3.453230619430542, 0.8868278861045837, -1.2714338302612305, 0.08037281781435013, 0.23792661726474762, 0.0031915539875626564, -0.28698408603668213, -0.6566457748413086, 0.15285348892211914, -0.8854640126228333, -0.33205142617225647, -0.900287926197052, -0.7689498662948608, 1.2431533336639404, 0.27951961755752563, -0.49661824107170105, 0.8993248343467712, 0.03591005876660347, 0.5267378687858582, -0.4750518798828125, -0.22654658555984497, 0.09701704978942871, -2.021427869796753, 0.00636289780959487, 0.17643797397613525, -0.9488822221755981, -0.7849218249320984, -1.4104443788528442, -0.6968591809272766, 0.8540253639221191, -0.08657371997833252, 1.1867659091949463, -2.160595417022705, 1.294793963432312, 0.7100968360900879, -0.13550090789794922, -0.22850175201892853, 0.005723981652408838, 1.42562735080719, -1.9049725532531738, -1.22329843044281, 0.6654671430587769, 0.1762928068637848, 0.679020881652832, -1.3950588703155518, 0.28586095571517944, 0.17797794938087463, -0.3691045641899109, 0.734181821346283, 0.46392008662223816, -0.8467825651168823, -0.10439138114452362, 1.287060260772705, -1.8833831548690796, 0.547740638256073, 1.8359893560409546, -1.2968833446502686], [1.1686142683029175, 2.474134683609009, 0.7117297649383545, -0.3235558867454529, 0.8782874345779419, 1.3536155223846436, 0.7898826003074646, -1.2705779075622559, -0.885259747505188, -0.7122266888618469, -0.09666546434164047, -0.1622304469347, 1.7850022315979004, -1.1596428155899048, 0.946890652179718, 1.193436622619629, -1.180016279220581, 0.273791640996933, -0.3741888105869293, -1.5084822177886963, 1.7061760425567627, 0.8623148202896118, -1.3691436052322388, -1.9616789817810059, -0.054647959768772125, 0.008619376458227634, 1.4965001344680786, 0.24473357200622559, 1.1569205522537231, 0.6499883532524109, 0.613152801990509, -1.264798879623413, 0.04089733585715294, 1.3520162105560303, 0.2181856781244278, 0.03690822795033455, 1.4472782611846924, 0.06432254612445831, -0.29446128010749817, 0.8233082890510559, 1.4543390274047852, 1.017356038093567, 0.8837094306945801, -0.9335389137268066, 0.8981414437294006, -0.0037057900335639715, 0.08333887159824371, -0.28618842363357544, 0.675000011920929, -1.3444807529449463, 0.46437397599220276, 0.265118807554245, 0.12131050229072571, 0.9882495999336243, 0.8578067421913147, -0.650418221950531, 0.23236970603466034, 1.530386209487915, -0.5048621892929077, -0.6840915679931641, -0.31207075715065, -0.7799407243728638, 1.0150463581085205, 0.12419538199901581], [0.1546265184879303, -0.41387343406677246, 1.1504814624786377, -0.3008826971054077, -0.6265310049057007, -1.8884400129318237, -0.23619569838047028, 0.4898039698600769, 0.8348497152328491, -1.3339253664016724, 0.07192030549049377, -0.9353735446929932, -0.4978501498699188, -1.1672331094741821, -0.123481385409832, 0.4016072452068329, -0.31631579995155334, 1.6055960655212402, -0.37644344568252563, -0.03549172729253769, -0.8561765551567078, 0.5753757357597351, 0.21993565559387207, -0.7392034530639648, 1.8566951751708984, -1.2460150718688965, 0.48735079169273376, 0.9170389175415039, 0.8489643335342407, 0.9682919383049011, -1.357746958732605, 1.4865198135375977, -0.22703443467617035, 0.015448637306690216, -0.8558114171028137, -0.5875241756439209, 0.27728626132011414, -0.4591505527496338, -0.08409936726093292, -0.20149242877960205, 0.3790324032306671, -0.8887317180633545, 2.421363115310669, -0.06504969298839569, -0.06814510375261307, -0.5644304752349854, 1.4085991382598877, 0.7081442475318909, -1.605811357498169, 0.6968793869018555, -1.6348201036453247, 0.8507328629493713, 1.3367598056793213, -0.7761673927307129, -0.049082860350608826, -0.36702272295951843, -0.8785494565963745, 0.9022380709648132, 0.3003789484500885, -0.7646687030792236, 0.4908977448940277, 0.6520406007766724, 0.5853375792503357, -0.08975617587566376], [-0.0037637141067534685, 1.0973498821258545, -0.46674972772598267, -1.6763604879379272, -0.3421933948993683, 0.8603608012199402, 0.12438514828681946, -0.28751376271247864, 1.9226654767990112, -0.9983465075492859, -1.6684321165084839, -1.0804176330566406, -0.5410722494125366, -1.0954288244247437, -0.1703394651412964, -0.1263190358877182, 0.17697176337242126, -0.8359006643295288, 0.5863766670227051, -0.4775421917438507, -0.7403879165649414, -1.8211090564727783, -0.16035179793834686, 0.980384349822998, -1.1238949298858643, -0.3511195778846741, 0.4614579975605011, -0.017421822994947433, 1.798020362854004, -1.088740348815918, -1.4186872243881226, -2.220721960067749, 0.41605469584465027, 0.6788917183876038, -0.9727544188499451, -0.36597418785095215, 2.984816074371338, 0.10063929110765457, -0.8810766935348511, 1.1986693143844604, 0.02082713320851326, 0.6272261738777161, -0.5623212456703186, -0.5447515845298767, 0.3446366786956787, -1.088680386543274, 1.4220318794250488, 1.3398555517196655, 0.5329686403274536, 0.19248363375663757, 0.3049968481063843, -0.9198966026306152, 1.2034157514572144, 0.15651248395442963, 0.4538770616054535, 1.9337382316589355, -0.9400902390480042, 0.010786646977066994, -1.1965324878692627, -1.1229623556137085, 0.47085192799568176, -0.3719094693660736, -0.37283286452293396, 1.86348557472229], [-1.5926448106765747, -0.995293378829956, 0.5479395985603333, 0.2224145531654358, -1.773419737815857, 0.6689217686653137, 0.4317767322063446, 0.24708852171897888, 1.102664828300476, -1.2448891401290894, 0.9655164480209351, 0.9322996139526367, 0.9824710488319397, -0.5622376203536987, -1.8612236976623535, 0.4248828887939453, 1.5386337041854858, -0.6688486337661743, -0.1094849705696106, -1.5221202373504639, 1.4278301000595093, -0.6155052781105042, -0.45800793170928955, 1.655218482017517, 0.2501988708972931, -0.9917988777160645, 1.8174186944961548, -0.8290839791297913, 0.7255167365074158, 1.3001972436904907, 0.24352838099002838, 0.12003032863140106, 0.04064619913697243, -0.18604908883571625, 1.6112221479415894, 0.7559948563575745, 1.671957015991211, -0.8492116332054138, -1.405074954032898, 0.6585540175437927, 0.9411081075668335, 0.38016971945762634, -0.9164614081382751, -1.5206124782562256, -0.6856281757354736, 0.17109397053718567, -0.045114532113075256, -0.8822148442268372, 0.23553846776485443, 1.846078634262085, -0.5303623080253601, 0.2322515994310379, 0.44854170083999634, 0.3369060754776001, 0.9364753365516663, 0.7507184147834778, 0.8837136030197144, -1.0457077026367188, 0.45085158944129944, -1.5023730993270874, -0.13531740009784698, -1.294723391532898, 2.239445924758911, 1.2731709480285645], [0.7929553389549255, 0.20551697909832, 1.8992396593093872, -0.9276694059371948, -1.6599739789962769, 0.44868409633636475, 0.24975839257240295, -0.4141421914100647, 0.7133038640022278, 0.6616501212120056, -0.10066057741641998, 0.9133851528167725, -0.4510742723941803, -0.646772027015686, -0.5737224221229553, -0.004918796941637993, 0.1724073737859726, 0.7586957812309265, 0.08039575070142746, 1.3531981706619263, 0.9502941370010376, -0.4313739836215973, -0.2348509579896927, -0.16660569608211517, 1.1930322647094727, -1.2337180376052856, -0.9717406630516052, -0.2545810043811798, -0.03348565474152565, 1.0566465854644775, -0.21491581201553345, -0.4408605396747589, -0.19055506587028503, 1.0894644260406494, -0.8574895262718201, 0.648842990398407, 0.25988680124282837, 0.1022706925868988, 0.8317028284072876, 1.3179646730422974, 0.245261088013649, -2.381362199783325, 1.1813077926635742, 1.305452585220337, -0.27135005593299866, 0.36794182658195496, -0.026482146233320236, -1.368608832359314, 1.864720106124878, 1.849027395248413, 1.3904786109924316, 0.049757130444049835, 0.21126914024353027, -1.2242703437805176, 0.172036275267601, -0.5008864998817444, 0.12323720753192902, -0.5618520975112915, 0.6171902418136597, -1.156677484512329, 0.3345346748828888, -0.1333865225315094, -1.7959132194519043, 1.1674258708953857], [0.03292863443493843, 1.3465298414230347, 0.4400666356086731, 0.7286524772644043, 0.5732355117797852, -0.8372763991355896, -0.7676871418952942, -0.3344348967075348, 0.5155213475227356, -1.0403950214385986, 0.2095096856355667, 0.7605487108230591, -1.26913583278656, -0.5024244785308838, 1.775712251663208, 1.356641173362732, -1.13187837600708, -1.1394588947296143, 0.7116896510124207, -0.3818904161453247, -0.28819939494132996, -0.20825643837451935, 0.3692290782928467, -0.25649961829185486, 1.9375020265579224, -0.5433260798454285, -0.32185447216033936, -2.159862995147705, 0.15879294276237488, -0.965617835521698, -1.5444289445877075, -2.7874765396118164, -0.617552638053894, -0.5278332829475403, 1.2631136178970337, 0.27830132842063904, -1.1998964548110962, 0.9029227495193481, 1.1300256252288818, 0.8753481507301331, 0.7232420444488525, 0.3453220725059509, -0.35624876618385315, 0.862207293510437, -0.139430433511734, -0.724231481552124, 0.430127888917923, -0.45820656418800354, 0.6444174647331238, -0.30978748202323914, 0.6851931810379028, -0.2593195140361786, -0.3984016180038452, -0.1956660896539688, 0.6784635186195374, 1.5724036693572998, -0.6019980311393738, 0.3571975529193878, 1.0777249336242676, 1.1127521991729736, -0.05735868215560913, -0.4298659861087799, -0.2155197560787201, 0.7384222149848938], [-0.27054303884506226, 0.2819668650627136, 0.7129194140434265, 1.3111562728881836, -0.6564967632293701, -0.07431512326002121, -0.5524539947509766, -1.8532065153121948, -0.6712080240249634, -0.7339985966682434, 0.5036023259162903, 2.2538607120513916, 0.7970514893531799, 1.150765061378479, 0.35171544551849365, -0.8752306699752808, 0.2074328362941742, -0.8627042770385742, 2.16839599609375, -0.34266743063926697, -1.318292260169983, 0.2799256145954132, -0.4244259297847748, -0.6797319650650024, -0.8995460867881775, 2.8072292804718018, 0.45460277795791626, -0.9963709712028503, -0.5302667617797852, -0.10642579942941666, 0.31002041697502136, -2.5823986530303955, -1.0499919652938843, 0.11601296067237854, -0.008245148696005344, -0.5352280139923096, 0.12364646792411804, -0.7168918251991272, 0.7394086122512817, 0.2886084020137787, 1.0932847261428833, 0.1724494993686676, -1.6668673753738403, -0.45021694898605347, -0.4030250608921051, 0.1981966495513916, 0.6002376079559326, 0.2969054877758026, 0.4877892732620239, 0.5321252346038818, -1.6042801141738892, 0.3192691206932068, -0.5446869730949402, 1.596949577331543, -0.3811517357826233, 0.9008297324180603, -0.4961247444152832, 0.1884596049785614, 3.0123484134674072, -0.17087911069393158, -0.7685258984565735, 0.074984110891819, -0.09304589033126831, 1.344560146331787], [-0.9353830218315125, -0.5862520337104797, 1.2788132429122925, 0.9191370010375977, 0.3028973937034607, -0.3297598958015442, -0.011236661113798618, 0.6438460946083069, 0.654434084892273, -2.1716415882110596, -1.0949405431747437, 2.3359928131103516, -0.882524311542511, -1.7250633239746094, 2.084195613861084, -0.22888703644275665, -1.0759631395339966, -0.4081200957298279, 0.5730751752853394, -1.0597823858261108, 0.9296137690544128, -2.0403034687042236, 1.1682326793670654, 0.44646385312080383, 1.9267497062683105, -0.09941104799509048, -0.6104593276977539, -0.08301038295030594, 0.014314359985291958, -1.2186267375946045, -0.07357217371463776, 1.5571188926696777, 0.6865144371986389, 0.7425602674484253, 0.15648764371871948, 0.7145897746086121, -0.19776809215545654, 0.7953384518623352, 1.102688193321228, -0.772081196308136, -0.7912218570709229, -0.8291415572166443, 0.8517318964004517, -1.1205884218215942, -0.9667479395866394, 0.5198253393173218, -0.7467924952507019, 1.077579379081726, 0.5264249444007874, 0.2568173110485077, -1.8343149423599243, -0.3913819193840027, 0.46252214908599854, 0.2396440953016281, -0.1593625247478485, -0.5688571333885193, 1.387932538986206, -0.6555518507957458, -0.17984013259410858, 0.7723436951637268, 1.0673396587371826, 0.1602233350276947, -2.242375135421753, -0.9162609577178955], [-0.2724878191947937, -1.3230457305908203, -0.13995422422885895, -1.6151716709136963, 0.651808500289917, 0.5681102871894836, 0.15628241002559662, 0.08943779766559601, 1.9590142965316772, -1.7500298023223877, -0.14826662838459015, -0.4789717495441437, -0.5975998640060425, 1.5623317956924438, 0.1015971377491951, -0.729089081287384, 1.798147439956665, 0.08416124433279037, -0.2619397044181824, -1.5126172304153442, -0.6475380659103394, -1.693705439567566, 1.6675188541412354, 2.2720119953155518, 1.7696443796157837, -0.8342357277870178, 0.4668843448162079, 1.3523218631744385, 0.7472894787788391, 0.3744088411331177, 0.6195815205574036, 0.35234931111335754, -0.9208129644393921, 0.17423051595687866, 1.220869779586792, 1.2009317874908447, -0.6266570687294006, -0.1712569147348404, 0.9014970660209656, -0.6036788821220398, 0.11391197890043259, 0.054594140499830246, -1.80819571018219, -0.4628623425960541, 0.8883972764015198, 0.6639021635055542, 1.1230617761611938, -1.1099486351013184, -1.4090418815612793, 0.11806543916463852, 1.261605143547058, -2.1153666973114014, -2.2012581825256348, 1.226885437965393, -1.3630954027175903, -0.7773452997207642, 0.5376521348953247, 0.8823330402374268, 0.9472434520721436, 0.5409125089645386, -0.1805105060338974, 0.7224915027618408, 1.0474846363067627, -0.5705879330635071], [-0.3161602318286896, -0.4472867250442505, 0.5232086181640625, -0.40303835272789, 1.54227876663208, -0.2772194445133209, 0.4500032067298889, -1.066978096961975, -0.2006554901599884, -0.7950016856193542, -1.8800280094146729, 1.0383085012435913, -0.08973269164562225, 0.7104909420013428, -0.051116570830345154, -0.48751941323280334, 1.6789350509643555, 1.2462201118469238, 1.1029983758926392, 0.4831527769565582, 0.362095445394516, 1.7412176132202148, 1.1607685089111328, 1.6564158201217651, 0.16372454166412354, -0.22947464883327484, -0.3621566593647003, -1.3357889652252197, 1.5465269088745117, -3.0784337520599365, -0.8096356391906738, -0.2151915431022644, 1.0894724130630493, -0.7223832011222839, 0.5575942397117615, 1.5362180471420288, 0.25318869948387146, 2.052032709121704, 1.311802625656128, 1.4730733633041382, 0.18980029225349426, 3.2901663780212402, 0.3502821922302246, 0.14805367588996887, -0.028204694390296936, -0.3307960033416748, 0.32201892137527466, -0.41876456141471863, -0.9193124771118164, 1.098746418952942, 1.191611886024475, -1.0377388000488281, 0.6970095038414001, -2.399055004119873, -0.07286843657493591, -1.2270731925964355, -0.6974355578422546, -0.3740358352661133, 0.3099207580089569, -0.6086311936378479, -0.2733649015426636, 0.17585770785808563, -0.2339058816432953, -0.5522187352180481], [1.512122392654419, 2.5848498344421387, -0.32939469814300537, 0.9683796763420105, 0.40039172768592834, -1.7969741821289062, -0.6447715163230896, 1.0069340467453003, 0.2569288909435272, 1.4695926904678345, -1.7293750047683716, 0.21202942728996277, 0.07549881935119629, 0.2621389627456665, -0.6165095567703247, 0.868636965751648, 0.925127387046814, -0.6973177790641785, 0.3242432177066803, -0.32223808765411377, -1.6918426752090454, 1.9394992589950562, 0.8508535623550415, 0.8499720096588135, -0.5763007998466492, -1.3868675231933594, 1.8750288486480713, -0.5112795829772949, 0.0995819941163063, -1.9265118837356567, -0.9772129654884338, -2.1742587089538574, 0.9727483987808228, 0.6443788409233093, -1.6877931356430054, 0.9505762457847595, 0.45892590284347534, -0.26564210653305054, -0.02222350612282753, 0.6781333684921265, 2.0373716354370117, -0.872391939163208, 1.2160382270812988, -0.8974732160568237, -0.32392075657844543, 1.2274247407913208, 0.16371314227581024, -0.4048421382904053, 0.697904109954834, -1.184651255607605, 0.9586762189865112, 0.18554991483688354, 0.9686301350593567, -0.6827327013015747, 0.11050955206155777, -0.7244179844856262, 1.7260370254516602, 1.249454140663147, -0.41634881496429443, -0.9358978867530823, -0.009551906026899815, 1.4061319828033447, -1.3702499866485596, -0.6899297833442688], [-0.3126987814903259, 0.16383394598960876, 0.061525218188762665, -0.21705053746700287, -0.6618691682815552, 0.8742081522941589, -0.0867900401353836, 0.5597889423370361, 0.5051921010017395, -1.239364504814148, -0.21471832692623138, 0.4679976999759674, -0.532171905040741, -0.800746738910675, -0.35315442085266113, -1.337227702140808, 0.8069459199905396, -0.11291952431201935, 0.4048417806625366, 1.067854881286621, -1.1345446109771729, -0.01200602576136589, 1.8426934480667114, -0.7950258851051331, 0.6643328070640564, -0.5907980799674988, 1.6587371826171875, -1.380411982536316, 0.1399122029542923, -1.6293773651123047, 0.7597894072532654, 1.2968522310256958, 0.18537336587905884, 0.45524725317955017, -0.007691897917538881, 0.1334402859210968, 0.32215452194213867, -1.524253487586975, -0.5433757305145264, -0.2108466923236847, -0.21199041604995728, 0.5396472215652466, 0.6857094764709473, 1.6192820072174072, -0.28738197684288025, 0.9356974363327026, 1.4873082637786865, 0.19166864454746246, -0.797169029712677, -0.9658324718475342, 0.10843969136476517, 1.0305240154266357, 0.8851299285888672, -0.6552140712738037, 3.0218312740325928, 0.611349880695343, 0.1495690643787384, 1.3245803117752075, 1.1279159784317017, 0.12635008990764618, 0.26674655079841614, 0.8097397089004517, -0.6726500391960144, -0.8631399273872375], [-0.04751311615109444, 0.40441209077835083, 0.38825076818466187, 0.1958545446395874, 0.23836439847946167, -1.6761806011199951, 1.3382025957107544, 1.0408662557601929, 2.52008318901062, 0.9112070202827454, -0.6353312134742737, -0.11971572041511536, -2.2119295597076416, -0.17460213601589203, 1.1975764036178589, 1.0675997734069824, -1.8344837427139282, 0.6566198468208313, -0.9758056402206421, -1.286721110343933, -0.27047204971313477, -1.4450076818466187, -0.08626928925514221, 0.4162156879901886, 0.3405231833457947, 0.4567447602748871, -0.43891504406929016, 0.2686130702495575, 0.8557100296020508, 0.7576348185539246, 1.4215350151062012, -0.1179666817188263, 1.0031094551086426, -0.15229712426662445, -0.3271515369415283, 0.12089338898658752, 0.33527126908302307, 1.7768306732177734, 2.0647923946380615, 0.39858677983283997, -0.34818702936172485, 1.7089853286743164, -0.038934964686632156, -0.34786751866340637, -1.3497861623764038, 0.26881834864616394, 1.1171504259109497, -0.48625507950782776, 0.31582075357437134, -0.3556344509124756, -0.012409511022269726, 0.18871919810771942, -1.4490848779678345, 0.35512837767601013, 1.3052797317504883, -1.3588542938232422, -0.7397551536560059, -1.6771785020828247, -0.9888585805892944, 1.663552165031433, -0.4766983687877655, -2.2470507621765137, 0.8873233795166016, -0.10276836901903152], [-1.0058562755584717, -0.16844742000102997, 0.1959158033132553, 0.941256582736969, 0.9376007318496704, -0.5291041731834412, -0.7138521075248718, -0.8433994650840759, -1.367209792137146, -0.6182641983032227, 1.1765789985656738, 1.7491544485092163, 0.10500434786081314, -1.4747766256332397, -0.11833177506923676, 0.5985926389694214, 0.5593895316123962, -0.17591245472431183, -0.8016645908355713, -0.1038181334733963, -0.1031288355588913, 1.0672221183776855, -0.5026815533638, -0.8030762076377869, 1.1935490369796753, 1.3626084327697754, 0.7813831567764282, 0.4241490066051483, -0.42156967520713806, 1.7686285972595215, 1.8482948541641235, -0.609806478023529, -0.06840875744819641, -0.8152293562889099, 0.004073583986610174, -0.6234119534492493, 0.9160842299461365, 0.5597731471061707, -0.6687965989112854, 1.2372357845306396, -0.14080794155597687, 0.7368096113204956, -1.2798060178756714, -0.2101498246192932, 0.36081284284591675, 1.1424448490142822, -0.3041027784347534, -1.0416864156723022, -1.3064934015274048, 0.10729631036520004, -2.6265721321105957, -0.6028584241867065, 1.4395923614501953, 0.4022088646888733, 1.1787257194519043, 0.036643337458372116, -0.9276779890060425, 1.3098413944244385, 0.5772010087966919, -0.29640114307403564, 1.960089087486267, -0.5014796853065491, 0.5304039120674133, -0.032747212797403336], [-0.15784364938735962, -0.2912808358669281, -0.0001692112855380401, -1.5074735879898071, 2.431804656982422, -0.8262630701065063, 0.5836182236671448, -0.6829367280006409, 1.2702298164367676, -0.27924028038978577, 0.2931319773197174, 1.1568278074264526, -0.57731032371521, -1.5566680431365967, 2.2297816276550293, 1.017585039138794, 0.17057931423187256, -0.06946955621242523, -0.7662285566329956, 0.18911337852478027, -0.8443856239318848, 1.0999871492385864, 0.7457864284515381, 0.9965798258781433, 0.8807547092437744, 0.9536153674125671, 0.29279467463493347, 1.180768609046936, 2.2262706756591797, -0.607316255569458, 1.3769726753234863, -0.4869269132614136, -1.7945373058319092, 0.957965075969696, -0.9164999723434448, 0.5794901847839355, 1.019131064414978, 0.13895685970783234, 0.9319255948066711, 0.12332812696695328, -0.5707554221153259, -0.5691444873809814, 0.38140740990638733, 0.7618275284767151, 0.16222041845321655, 0.9843397736549377, 0.02341882511973381, 0.5673362612724304, -0.653965950012207, 0.5379043817520142, 1.925298810005188, -0.6433067917823792, 0.3142222464084625, 0.45756757259368896, -0.7314046025276184, 0.5417049527168274, -0.9835756421089172, 0.7063552737236023, 0.3450947701931, 0.9191470146179199, -0.07974579930305481, -0.4856705069541931, 0.3812965154647827, -0.13045033812522888], [0.5751351118087769, 0.9999446868896484, -0.3096858263015747, -0.4496913552284241, 0.3179134130477905, -0.5869809985160828, -1.2485344409942627, -0.32490673661231995, 0.7468354105949402, 1.895186424255371, 0.9609255194664001, 1.7506524324417114, -0.6907494068145752, -1.0786168575286865, -1.424316167831421, 0.1386168897151947, -1.2299047708511353, -1.3190951347351074, 1.6841199398040771, -0.20177221298217773, 0.10176999121904373, -0.12873682379722595, 0.9309653639793396, -0.020758409053087234, 1.082943320274353, -0.16289223730564117, 0.30699288845062256, -1.368854284286499, -0.23074223101139069, -0.948206901550293, -0.2148958295583725, 2.390246629714966, 0.9804477691650391, -1.3127647638320923, -0.9466044902801514, 1.8154304027557373, 0.10169417411088943, -0.008817439898848534, 0.5323712825775146, 1.196244239807129, 0.09614673256874084, -0.2928575873374939, -0.9969547986984253, -0.09008621424436569, -0.1120525673031807, -0.1234736517071724, -0.656880259513855, -0.6302370429039001, 1.3237652778625488, 1.469456434249878, -2.1113831996917725, 0.7283506393432617, -0.22115212678909302, -0.00519301975145936, 0.12895622849464417, -1.6432794332504272, 0.3524129390716553, 0.4803628921508789, 0.449029803276062, 1.7478718757629395, -1.8673619031906128, 0.9106568694114685, -1.5334311723709106, -0.3503318428993225], [-1.1630407571792603, -0.010209773667156696, -0.4107518196105957, -0.12883321940898895, -0.8985362648963928, 0.4009646773338318, 0.797548770904541, -0.7378438711166382, -1.3750966787338257, 0.4350629150867462, -0.5375925302505493, 1.0346938371658325, 1.9095661640167236, 0.504683792591095, 1.3477344512939453, -0.474422812461853, -0.7795848846435547, -2.02751088142395, 0.07563000172376633, 0.05305611342191696, -0.47526276111602783, 0.13473473489284515, 1.5630342960357666, 0.9042973518371582, 1.0901893377304077, 0.4163203239440918, -0.21179628372192383, 1.5893752574920654, 0.13017790019512177, 1.340079665184021, -0.5760068893432617, -1.0067399740219116, 0.49280741810798645, 0.9335537552833557, -1.1304315328598022, 0.21704313158988953, 1.2438910007476807, -0.016878390684723854, -0.30173438787460327, 1.0765198469161987, 0.48671630024909973, 0.9578773975372314, 1.313287377357483, 0.5840935111045837, 1.1107500791549683, -0.4606313407421112, 0.6541959047317505, -1.5118988752365112, 0.8982948064804077, 0.4695534408092499, -0.9118029475212097, -1.072695255279541, 0.8090202212333679, 2.3005564212799072, -0.8646736741065979, 1.848956823348999, 0.5897641181945801, -1.4982869625091553, 1.945309042930603, 0.06347605586051941, 1.1079972982406616, 1.0701392889022827, -1.5908362865447998, 0.19638293981552124], [0.02008468098938465, 0.4781006872653961, 0.7989062070846558, -0.9524474740028381, -1.3982816934585571, -1.4519752264022827, 0.22596998512744904, -1.3148658275604248, 0.4427899122238159, -0.6178662180900574, 0.057839591056108475, 0.12019132822751999, 1.2908074855804443, -1.2338054180145264, 0.6129831671714783, 1.584428071975708, 0.3247654438018799, -1.357520580291748, 1.5510330200195312, -0.20519882440567017, -0.8377276659011841, -1.3806630373001099, -1.1420897245407104, 0.41603103280067444, 2.1315083503723145, -0.6149755120277405, -0.7448462247848511, 0.05872771888971329, 0.6920273303985596, -0.5513780117034912, 0.12819242477416992, -0.5441139340400696, 0.3217262923717499, 0.22045396268367767, -1.1757317781448364, -1.3177448511123657, 0.7915298342704773, 0.2945990562438965, -0.3663339614868164, -0.3863053619861603, 0.7386800646781921, 0.26750242710113525, 0.3417644500732422, 0.28037652373313904, -0.7185286283493042, 0.018978966400027275, 0.837218165397644, -0.4802057147026062, -1.2237824201583862, -0.0301449503749609, -0.6859308481216431, 0.6150825023651123, -0.32750609517097473, 0.39193594455718994, 0.37326180934906006, 0.3853406608104706, -0.7622013688087463, 0.31105220317840576, 1.3049899339675903, -0.24181492626667023, -0.7900692820549011, -0.5524833798408508, -0.7692705392837524, -1.0808967351913452], [0.6664791107177734, -0.730089008808136, 1.2504165172576904, 1.0518207550048828, 0.5849964618682861, -0.042194366455078125, 0.9981880187988281, 0.43205884099006653, -0.702440619468689, 0.9103719592094421, 0.6473779082298279, -0.42134326696395874, -0.04495079442858696, -1.0605336427688599, 0.38009321689605713, 0.6505184173583984, -0.42000052332878113, -1.7911083698272705, 0.3485139012336731, 0.3026975393295288, -1.6700307130813599, -0.38225579261779785, 0.9205664992332458, -0.8523365259170532, 0.15512169897556305, 1.8541651964187622, 0.0796230360865593, 0.34868761897087097, -1.3887948989868164, -0.1555718034505844, -0.6695775389671326, 0.119715616106987, 0.28320133686065674, -0.3574540615081787, -1.3237261772155762, -0.018156468868255615, -1.0673261880874634, 1.9009438753128052, 1.2814443111419678, 0.2135043740272522, 1.1421452760696411, -1.7488908767700195, 0.20168228447437286, -0.5544900894165039, 1.3184305429458618, 0.1027604416012764, 0.412069171667099, 1.938926100730896, 0.2620697021484375, -0.2643528878688812, 0.20022451877593994, 0.3043162226676941, 0.3759491443634033, 0.1740814745426178, 1.3971259593963623, 0.5238417387008667, 0.17881684005260468, -2.0293262004852295, 0.910803496837616, 0.408126562833786, 0.38047805428504944, 0.31876716017723083, 0.8924763798713684, -0.37787267565727234], [1.0571955442428589, 0.14462898671627045, -0.2451806366443634, -1.2256946563720703, 0.6689661741256714, -0.5406421422958374, -0.6227654814720154, -0.2202339768409729, 0.08604594320058823, 2.330881118774414, 0.17094218730926514, -1.912580132484436, -0.3726442754268646, 0.7010209560394287, 0.6588956117630005, -0.5566980242729187, -1.277154564857483, 0.06126965582370758, -0.5819854736328125, 0.3519507646560669, 0.7703834176063538, -0.08477288484573364, -0.21990591287612915, 0.32909929752349854, -1.5026800632476807, -0.15105487406253815, 0.7346669435501099, 0.07372602075338364, -1.360615611076355, 0.35216057300567627, 1.0578027963638306, -0.36468684673309326, 0.6730824708938599, -0.5835525989532471, 0.30880892276763916, -0.8429375290870667, 0.35222354531288147, -1.5559322834014893, 0.5012078881263733, -0.11587344855070114, -1.0235782861709595, -0.1515704244375229, 1.057462453842163, 0.6543483734130859, 0.6481068730354309, -0.44313693046569824, 0.6848170757293701, -0.3147435188293457, 0.7388980984687805, -0.42912009358406067, -0.16175682842731476, 0.27998530864715576, -1.5699961185455322, 0.5096498131752014, -0.2570969760417938, 1.2861855030059814, -1.0714731216430664, 0.5672884583473206, 1.2254475355148315, 1.2357828617095947, -1.1280938386917114, 1.146714687347412, 0.06065551936626434, -0.9350690841674805], [-0.7948066592216492, -0.7617694735527039, 1.7507613897323608, 0.9469054341316223, -0.1203216090798378, 0.4402111768722534, -1.5311845541000366, 1.1335023641586304, 0.22443903982639313, 0.45080089569091797, -0.07222440838813782, -0.6838563680648804, -0.8273711800575256, 1.690434455871582, -0.03314361348748207, 0.5676511526107788, 0.48416227102279663, 0.8900855183601379, 0.47353464365005493, -0.6877214312553406, 1.26587975025177, -0.7561372518539429, -0.09390321373939514, 0.5601438879966736, -0.5922562479972839, -1.7874629497528076, 0.19724953174591064, 1.3054639101028442, -0.38697296380996704, 0.39234858751296997, 0.028038037940859795, -1.0134714841842651, -0.3676483929157257, -0.4709286093711853, 0.8187525272369385, 0.42168721556663513, -0.20251208543777466, -1.5787649154663086, 0.2993929386138916, 0.26981765031814575, -0.827362060546875, -0.4927865266799927, 2.114409923553467, 1.2154736518859863, 0.464388906955719, 0.2226228266954422, -0.8013516664505005, -0.6479538679122925, -1.5548698902130127, -0.034982841461896896, -2.466139078140259, -0.32125240564346313, 2.4821393489837646, -0.44064345955848694, -0.7858406901359558, -1.201644778251648, -0.5554553270339966, -0.6941049098968506, -1.0439141988754272, -0.6128289103507996, 0.40772679448127747, -0.01569330506026745, -2.897256374359131, 2.1911749839782715], [-0.9252955317497253, -0.41113561391830444, -1.31683349609375, 0.6763556599617004, 0.4414568245410919, 1.7521157264709473, -1.6429848670959473, -0.7824515700340271, -0.06957697123289108, 0.12651070952415466, -0.2811271846294403, 1.0406721830368042, 0.6257473826408386, 0.5733075141906738, -0.05040189251303673, -0.39457404613494873, 1.2537773847579956, 0.5570365190505981, 0.3226979970932007, 0.05485142022371292, 0.8571973443031311, 0.36890700459480286, 1.4999676942825317, 0.37369272112846375, 0.44465431571006775, -0.4506540298461914, -1.1503355503082275, 0.4228276312351227, -0.23415423929691315, 0.7729991674423218, 1.6918070316314697, 0.3645261526107788, 0.7654901146888733, 2.141094446182251, -1.423399567604065, -0.9004576206207275, 1.0943135023117065, -0.7459506988525391, 0.3643336892127991, -2.3308897018432617, -0.3475380837917328, -0.8416175246238708, -0.5509918928146362, 0.10844843089580536, 0.3644084930419922, -1.3905301094055176, -2.1421163082122803, -1.4510291814804077, 0.4795178472995758, -0.21576164662837982, -0.5976874828338623, -0.8102750182151794, 1.0994560718536377, -0.004355454817414284, -1.5160294771194458, 1.3355566263198853, 0.7686769366264343, -1.387690782546997, 1.5878771543502808, -2.0329349040985107, 1.0382604598999023, -1.4011247158050537, -0.8191335797309875, -0.6799460649490356], [-1.1538097858428955, 3.4237375259399414, 0.4345371425151825, -0.25365638732910156, 0.4259306788444519, -1.0987491607666016, 0.61748868227005, -0.7638448476791382, -0.13098832964897156, -0.901898980140686, 0.10762328654527664, -1.5812203884124756, 0.39399254322052, 0.08999136090278625, -0.09631609171628952, 0.21845118701457977, 0.01312168873846531, 0.13279256224632263, 0.09501438587903976, 0.5207509398460388, -0.6897377371788025, 1.3194189071655273, 0.28698238730430603, -1.0604898929595947, -0.8876616358757019, -0.7123284935951233, -1.0812525749206543, 1.3869776725769043, -0.534918487071991, 0.3088625371456146, -0.6588620543479919, -1.4330497980117798, 0.6433008909225464, -0.4724960923194885, -1.0535695552825928, -1.0473605394363403, 0.4797935485839844, 0.37626779079437256, 0.7465904951095581, 0.44813549518585205, -0.005061136558651924, -3.2002134323120117, 0.41715604066848755, -0.781495213508606, 0.9033604860305786, -0.6207078099250793, -0.65975022315979, 0.2599542438983917, 0.6243516802787781, 1.2132562398910522, -0.9660797715187073, -0.02647078037261963, -0.15541298687458038, 0.699711263179779, 0.6743633151054382, 0.05713443458080292, 1.6643322706222534, -0.39309215545654297, 1.5320218801498413, -0.6997703313827515, 0.20765483379364014, -0.02809702791273594, 1.0818771123886108, 2.245957612991333], [0.6371183395385742, 0.2805041968822479, -0.8965059518814087, -0.4149557054042816, 0.26914042234420776, 0.2991831600666046, -0.7300942540168762, 0.8746829628944397, -0.5434345006942749, 2.209796667098999, -0.7502378821372986, 0.8652455806732178, -0.23683828115463257, 0.05072661489248276, -1.2610764503479004, 2.0098764896392822, -0.7076570987701416, -0.43160533905029297, 0.652675449848175, 0.5696833729743958, -1.9461164474487305, 0.6564006209373474, -0.09544436633586884, 0.884775698184967, -1.0313310623168945, 0.21218504011631012, 2.217592239379883, 1.2321304082870483, 0.024704450741410255, -0.27298036217689514, -2.2544353008270264, -2.6160995960235596, -0.05443583428859711, -0.7079006433486938, -0.9446355700492859, 1.5066756010055542, -1.1290569305419922, -0.8874054551124573, -0.4316888153553009, -0.475259006023407, -0.3055485785007477, 0.5841178894042969, -0.17064052820205688, -1.0569918155670166, 0.9128270149230957, 1.6132687330245972, -1.8901596069335938, 0.1030958890914917, -0.03609203174710274, -0.3093700110912323, 2.0752358436584473, 0.18604151904582977, -0.14792777597904205, 0.05029459297657013, -0.7520763874053955, -0.5138957500457764, 0.07547849416732788, 0.23828847706317902, 0.974412739276886, -0.43628162145614624, 0.7067601084709167, 1.2400418519973755, -0.43264082074165344, -0.7519471049308777], [0.4261627793312073, 0.7224077582359314, -0.06307191401720047, 0.6531550288200378, -0.031979288905858994, -1.251720666885376, -0.4003775715827942, 0.3571769893169403, -0.04819100722670555, 1.0610469579696655, 1.138123869895935, -1.9446024894714355, 0.48291099071502686, 0.2915794849395752, -0.4013778567314148, -0.33930671215057373, 0.19136056303977966, 0.9681891798973083, -0.27572768926620483, -0.2974691689014435, 0.07986485213041306, -0.19446155428886414, 0.21217778325080872, -1.5301308631896973, -0.47504913806915283, 0.43596526980400085, 0.8354210257530212, 0.5534940958023071, 0.23684175312519073, 0.8041726350784302, -0.24808938801288605, -0.40946584939956665, -0.020141825079917908, 0.20331382751464844, -2.0762221813201904, 0.34415945410728455, 1.849068284034729, -0.21903526782989502, 0.019640028476715088, 0.39196982979774475, -0.02457890287041664, 1.125447154045105, 0.6700132489204407, 0.008457795716822147, 0.9186272025108337, -0.5063967704772949, -0.7734358906745911, 0.6593895554542542, 1.6006040573120117, -2.823636293411255, -1.4383426904678345, -0.35909879207611084, -0.11353136599063873, -0.1346050202846527, 0.6220420598983765, 0.7879798412322998, -0.7862955331802368, 0.9605841040611267, -1.0552406311035156, -0.4128301739692688, 1.083736777305603, -0.24243903160095215, -0.059464745223522186, -0.09002026170492172], [1.8135334253311157, -0.6680117845535278, 0.012249651364982128, 0.9104326963424683, -0.23643681406974792, -0.025555923581123352, 0.5501725077629089, 2.0157012939453125, 0.6158725023269653, -0.2526404559612274, 1.6709327697753906, 2.2634170055389404, 1.0646700859069824, -1.2511111497879028, -2.202176570892334, 2.4116132259368896, 1.5967772006988525, -0.7209205627441406, 0.10437613725662231, 1.2733614444732666, -1.6232818365097046, -0.0915631502866745, 0.9493467211723328, 0.337863564491272, -0.26013514399528503, -0.239587664604187, 1.6260628700256348, 1.321515440940857, -0.6126768589019775, 0.39272984862327576, -0.5818194150924683, -0.5712961554527283, 0.9633839130401611, -1.3334910869598389, 1.52737557888031, -1.5156514644622803, -0.20372051000595093, 0.45870646834373474, -0.797723650932312, -0.6943209171295166, 0.45347127318382263, -0.3112967610359192, -0.07621337473392487, 0.5018637180328369, -0.6764399409294128, -0.29945671558380127, 1.1007035970687866, -1.4479553699493408, 0.2005937099456787, 0.2814522683620453, 0.08010516315698624, -1.7955102920532227, 0.20295237004756927, 0.1551787406206131, -1.2599983215332031, 0.7307984232902527, -0.9218027591705322, 0.5635786056518555, -1.2675564289093018, -0.5934221148490906, -0.17660394310951233, -0.469196617603302, 0.6533408164978027, -0.012248974293470383], [-0.411546915769577, -0.4736899733543396, 0.2912517786026001, -0.4213562607765198, 0.676247775554657, -0.42353805899620056, -0.6481841802597046, 0.1942570060491562, -0.1200423464179039, 1.4383028745651245, -0.17873449623584747, -1.1693880558013916, -0.32606813311576843, 0.08513958752155304, 0.883015513420105, 0.1673000305891037, -1.1599756479263306, 0.2500661611557007, 0.7636099457740784, 2.2960991859436035, 0.8740384578704834, 0.713858425617218, 0.23142628371715546, 0.02158251963555813, 0.4241686463356018, -1.1678560972213745, -0.2705685794353485, -0.3312790095806122, 0.4490436911582947, -0.1721799075603485, 0.011495624668896198, -0.6766232252120972, -0.7565160989761353, -2.145098924636841, 0.08850841969251633, -0.7975598573684692, 0.24336425960063934, -1.8553463220596313, -0.3141695559024811, -0.04779485613107681, 0.002532802289351821, -2.0885863304138184, -0.6343455910682678, 0.24319201707839966, 1.509150743484497, -0.30508437752723694, 0.7752997279167175, 2.062725067138672, 0.7323557138442993, -0.13618338108062744, -1.8993220329284668, -2.063638210296631, -0.2928241789340973, 0.6581079959869385, -1.0811594724655151, -0.37540459632873535, -0.19996555149555206, -1.8291049003601074, -0.9289883375167847, 0.3598002791404724, -0.02729859948158264, -0.13051822781562805, -1.0331844091415405, -0.49988874793052673], [2.3272879123687744, -0.6716786026954651, -0.9623684883117676, -1.2212848663330078, -0.15114633738994598, 0.7023056745529175, -0.1624460518360138, -1.6547187566757202, 0.6191036701202393, -0.7521912455558777, 0.1790013313293457, -0.9866795539855957, -0.4029756188392639, 1.606684923171997, -0.06260403990745544, -0.09106991440057755, -1.807785987854004, -1.4056297540664673, 0.45678815245628357, -0.41822361946105957, -0.6754481196403503, 1.0625782012939453, 0.2943928837776184, 1.4593095779418945, -0.1518651694059372, -0.12456785142421722, -0.8347750902175903, -0.7414745092391968, -0.1797093003988266, 0.10533417761325836, 0.301461398601532, 0.3522091209888458, 0.3412179946899414, -0.49597859382629395, 0.8562942147254944, 1.1464552879333496, -0.12708254158496857, 0.21987125277519226, -1.2550350427627563, -0.6488566994667053, 0.2913937568664551, 0.7025883793830872, -1.0873922109603882, 1.0911885499954224, -1.1457115411758423, -0.220501109957695, -0.5363181233406067, 0.8096163272857666, 0.40815141797065735, 0.9075019955635071, 0.8052291870117188, 0.4284428358078003, -1.2537071704864502, 0.11853343993425369, 0.1267702281475067, -0.2376977652311325, -0.5343102812767029, 1.0875399112701416, 0.13031181693077087, 0.5230847001075745, -0.8528487682342529, 0.06756393611431122, -0.3592294752597809, 0.7137270569801331], [1.943806529045105, 0.628510057926178, 0.8268754482269287, -0.0977463647723198, 0.3628540337085724, 0.7997351288795471, 1.0670069456100464, 0.43839406967163086, -0.7907328605651855, -0.7436694502830505, -0.6695062518119812, 0.7092637419700623, 0.5122133493423462, 0.7384718060493469, -1.0366566181182861, -0.8799794912338257, 0.10931463539600372, 0.14916115999221802, -0.6261898279190063, -0.9876188039779663, -0.056721996515989304, 0.7349601984024048, -1.4973214864730835, -0.502720296382904, 2.217982053756714, 1.1557542085647583, -0.1507047712802887, -0.5264077186584473, -0.9641813039779663, 0.35465916991233826, 0.506382942199707, 1.0169392824172974, -0.5220113396644592, -1.4615075588226318, 0.35541531443595886, -0.30430641770362854, -0.10181478410959244, -0.5068166851997375, 0.15394100546836853, 1.9866069555282593, 0.1762496531009674, 0.6176748275756836, -0.27877962589263916, -0.5386183261871338, -0.04794735461473465, 1.0644375085830688, 0.49983033537864685, -1.9231512546539307, -0.7133955359458923, -0.23114553093910217, 0.42400941252708435, -1.1079984903335571, 0.3302980065345764, -0.34147804975509644, 0.6629836559295654, 0.5740890502929688, 0.32007598876953125, 0.5909717679023743, -1.675478458404541, -1.0090587139129639, -0.08875944465398788, -2.761084794998169, -0.44936028122901917, 0.07697387039661407], [1.0505528450012207, 1.087274193763733, -1.0694580078125, -0.2744535505771637, -1.8555392026901245, -1.5143367052078247, 0.600306510925293, -0.1470762938261032, 0.4223608672618866, 0.7827589511871338, 0.4896022379398346, 0.27679380774497986, 1.2287802696228027, 1.082004189491272, -0.5665621757507324, -0.3584778606891632, 0.4700126051902771, -0.5011038780212402, 0.2789579927921295, -0.5437340140342712, -0.9287786483764648, -0.6812511086463928, 1.8977794647216797, -0.7724112272262573, -0.5174834132194519, -0.24675485491752625, -0.35458359122276306, -2.769542932510376, 0.6082716584205627, 0.9828938841819763, -0.14986787736415863, -0.1406814455986023, -0.8388334512710571, -0.5516883134841919, 3.0571210384368896, 1.2603154182434082, 0.07170303910970688, 0.8973972201347351, -0.3867833614349365, -1.531406283378601, 0.33488595485687256, 1.8114196062088013, -0.8944410681724548, -1.4718210697174072, 0.2053622007369995, -2.4194202423095703, 1.4585275650024414, 1.4703829288482666, -0.4616309404373169, 0.8907155990600586, -1.7101023197174072, -1.2188969850540161, -1.0565582513809204, 1.2115373611450195, -2.137047052383423, -0.09723509103059769, 0.9034962058067322, -0.46515563130378723, 1.8801393508911133, -0.5568246245384216, 0.3935370147228241, -0.2621200382709503, -0.7573188543319702, 0.5697532296180725], [0.6150599122047424, -0.27030396461486816, 0.38050422072410583, 1.190998911857605, -1.8242642879486084, 1.0404820442199707, -1.825649380683899, 2.4403622150421143, -2.0709517002105713, -1.9623708724975586, 0.44137758016586304, -0.8841420412063599, -1.8233705759048462, 1.243171215057373, -1.7537871599197388, -0.10163282603025436, 0.08100657910108566, 1.559605598449707, -0.8644454479217529, -1.6516777276992798, -0.10428524762392044, -0.34515947103500366, 2.566800355911255, 0.038377635180950165, -0.3446256220340729, -0.4746798276901245, -0.7798643112182617, 0.5057945847511292, 2.4176528453826904, -0.6550613641738892, 0.4497656524181366, 0.5319059491157532, 0.9964203834533691, -1.467108130455017, -0.6319224834442139, 1.37167227268219, -1.7416430711746216, -0.38795822858810425, 0.13806281983852386, -0.6715089678764343, -0.871471107006073, 0.4682231843471527, 0.19633585214614868, 0.6320352554321289, -1.9267889261245728, -0.7396988272666931, -1.0197805166244507, 2.1152536869049072, -2.0868799686431885, 0.5849772691726685, -0.18081547319889069, -1.8185076713562012, 1.6022406816482544, 0.00584757886826992, -0.7681989669799805, -1.1766459941864014, -0.47681811451911926, 0.3298376798629761, 0.5215465426445007, -0.7321103811264038, -1.090532660484314, -1.6045082807540894, -0.25426167249679565, -1.4502218961715698], [-1.6836320161819458, 1.2263444662094116, -0.008870396763086319, -1.519121766090393, 0.5139182806015015, -0.38554996252059937, 0.01690460555255413, -0.5016406774520874, -0.49304449558258057, 0.1595831662416458, 1.455734133720398, -0.6373173594474792, 0.015738138929009438, -0.509967029094696, -0.05948588252067566, -0.12269669026136398, -0.8862526416778564, -3.1857051849365234, -0.6831942796707153, -1.004775047302246, 1.7057716846466064, 1.6776498556137085, 0.016968071460723877, -0.9690112471580505, -0.7575091123580933, 1.9680571556091309, -0.5036224126815796, -1.9458274841308594, 0.2768152356147766, -0.9602656960487366, -1.6726100444793701, -0.232586070895195, 0.08264748007059097, -0.2722918689250946, 1.0022647380828857, -0.024901248514652252, 0.01888253726065159, 0.13781745731830597, -1.174867868423462, 0.24646499752998352, -1.4364771842956543, -1.4305400848388672, -0.6679956912994385, 0.0930023342370987, -0.3870064914226532, -0.027256907895207405, 0.549285888671875, 0.16630303859710693, 0.6707531213760376, -0.06433330476284027, 0.516297459602356, 0.1671736240386963, -1.7461144924163818, 1.59641432762146, -0.5152197480201721, -0.13448785245418549, 0.2740616798400879, -1.5572081804275513, 1.616593599319458, 0.9809110760688782, 0.7857839465141296, -1.2016828060150146, 1.2983112335205078, 0.2154264748096466], [-1.4398279190063477, 0.612310528755188, 0.7395062446594238, -1.0937540531158447, -0.8310019373893738, -0.4181196689605713, -0.36639779806137085, 0.2536431550979614, -0.05182589590549469, 0.7536594867706299, 1.1455777883529663, 1.0740145444869995, 0.4042394161224365, 0.2484990358352661, -1.7182269096374512, 0.292756050825119, -0.3242170512676239, -0.26030781865119934, 1.0809401273727417, 0.233791321516037, -1.013880968093872, -0.6532942652702332, 0.1706089973449707, 0.4644877016544342, 0.4990856945514679, 1.486751914024353, 1.0028284788131714, 0.9917821288108826, 0.1895202249288559, -0.4821804165840149, 0.26475846767425537, 0.5687596797943115, 1.083417534828186, -2.63808274269104, 0.6286739706993103, -0.8743276596069336, -0.5025882720947266, 2.0525403022766113, -0.10990319401025772, 1.9768038988113403, 0.4065417945384979, 0.9036871790885925, -0.5769202709197998, -0.866942286491394, -1.4461805820465088, 0.9557991623878479, 0.8859331607818604, -0.20720168948173523, -1.4877164363861084, 0.45858603715896606, -0.1506914496421814, 3.3701038360595703, -0.08535050600767136, 1.8135924339294434, 1.3554601669311523, -1.3031246662139893, -0.09300673007965088, -0.40697363018989563, -0.840950071811676, -0.32470518350601196, 0.4547482430934906, -0.0236513651907444, 0.0051544103771448135, 0.007099224720150232], [0.5539045929908752, 0.8119728565216064, -0.20284299552440643, -2.359229326248169, 1.02121102809906, 0.20054315030574799, 0.9660288691520691, 0.3491666615009308, 1.691983699798584, -0.10751090943813324, 0.110629603266716, -0.09723453968763351, 0.9575033187866211, -1.61976957321167, -0.015315576456487179, 0.5705674886703491, 1.534658670425415, 1.1277070045471191, -1.2676942348480225, -0.14316989481449127, 0.1191902756690979, 0.3388538956642151, 1.502659797668457, -1.6877766847610474, 0.7550838589668274, 0.2788969576358795, -2.1075439453125, 0.7604995369911194, 0.2261333167552948, -0.6156863570213318, -0.10372886061668396, 2.147494077682495, -0.44845688343048096, -0.1414988785982132, 1.7010586261749268, 0.5295270681381226, -0.45774734020233154, -1.932254672050476, 0.8411284685134888, 0.37565547227859497, 0.518346905708313, 0.4584859311580658, -0.5437504053115845, 1.008301854133606, -0.6478105187416077, -1.7790149450302124, -1.309065818786621, -1.8321782350540161, -1.3576279878616333, -0.5845929384231567, 0.34908327460289, 0.12561464309692383, 2.138333320617676, 1.1855535507202148, 0.062311891466379166, 0.5351371765136719, 1.053749680519104, -0.13883058726787567, -1.3439762592315674, 2.9328677654266357, 0.18954317271709442, -0.4132935106754303, -0.6376426815986633, 0.5614630579948425], [-0.23689904808998108, -0.6728000640869141, -0.8212996125221252, -1.31770920753479, -0.2680860161781311, 0.38273006677627563, -0.1984270215034485, -0.7393583655357361, -0.6758402585983276, 0.2501213848590851, 0.4234638512134552, 0.9134458303451538, -1.3981674909591675, -0.6805184483528137, -2.1932718753814697, 0.5570405125617981, -2.0363757610321045, -0.07335659116506577, 0.4080130457878113, -0.169759601354599, -1.1959354877471924, -0.6767407059669495, -0.9558364152908325, 1.3989224433898926, 0.4868159294128418, 1.6022766828536987, -0.5766329169273376, -0.9929060935974121, -0.49452146887779236, -1.0484563112258911, 1.4188621044158936, -1.6877224445343018, -0.023836230859160423, 1.3955109119415283, 0.49945420026779175, 2.3413636684417725, 0.5594816207885742, -0.645611584186554, -0.16946764290332794, 0.018390260636806488, -0.18423029780387878, -0.8711163401603699, -0.5123035311698914, 0.1475660651922226, 0.4766244888305664, -0.16755230724811554, -0.5916225910186768, -1.9464821815490723, 1.5435287952423096, -0.3998427093029022, 0.7963144779205322, 0.7330652475357056, 1.755563735961914, 1.337773084640503, 0.3357890546321869, 0.7028049826622009, 0.06058351323008537, 1.2395365238189697, -0.4950850009918213, 0.739677906036377, -0.05247955396771431, -1.4646588563919067, 0.030031196773052216, 1.0128710269927979], [0.8352905511856079, 2.294280767440796, -0.902106761932373, -0.6085097193717957, 0.02090575359761715, 1.0179777145385742, 0.6129130125045776, 0.14409597218036652, -0.67489093542099, -0.32036349177360535, -1.2172489166259766, 0.06687254458665848, 0.9216652512550354, -0.051792699843645096, -0.8465184569358826, 0.5918713212013245, 0.9614247679710388, 0.6961610913276672, -0.44503337144851685, 0.41078171133995056, 0.4552668631076813, -0.5478058457374573, -1.984433650970459, 0.5822174549102783, -0.2668672800064087, -0.749387800693512, 1.0539960861206055, -1.0944006443023682, 0.3931576609611511, 0.5259788036346436, -0.42423996329307556, 0.4204379916191101, -0.312747061252594, -0.9252780675888062, -2.4328086376190186, 0.2493806928396225, 2.3580708503723145, -0.5355210304260254, 0.5778291821479797, -1.720798373222351, 0.043827202171087265, 0.13617391884326935, 0.7148650884628296, 0.05129937827587128, 0.27365046739578247, 0.4045829176902771, 0.3385009467601776, 0.11074928939342499, 0.8584519028663635, -1.5302542448043823, -1.3899351358413696, -1.4911731481552124, 1.1935228109359741, 1.1543240547180176, 1.1621294021606445, 0.559313178062439, -0.6071140170097351, 1.4358781576156616, 2.4885973930358887, -0.7784892320632935, 0.273811936378479, -0.35781463980674744, -1.945878505706787, -0.4819098711013794], [-0.6437616944313049, 0.4639063775539398, 2.326972484588623, -1.3130234479904175, 1.562196969985962, 0.8269398808479309, 2.604020833969116, 0.9385948181152344, -0.17922750115394592, -0.17030738294124603, 1.2368383407592773, -2.3262884616851807, 0.27141040563583374, 1.5082776546478271, 0.3137105703353882, -2.7742691040039062, -1.652026891708374, 0.43140992522239685, 0.003252861090004444, 0.1090785339474678, -0.8553116917610168, -1.2883930206298828, 0.9201962947845459, 0.8455066084861755, -0.36277636885643005, -1.7233715057373047, -0.19805918633937836, -0.6918794512748718, -0.828393280506134, -0.07360346615314484, 0.975160539150238, 0.7818242907524109, 0.8158814311027527, -0.3516615331172943, 0.4407508969306946, 0.8704813122749329, 0.30102983117103577, -0.8987055420875549, 1.1471055746078491, 0.648049533367157, 0.5543057322502136, 1.4947113990783691, -0.11920329928398132, 0.5283119082450867, -0.7764819860458374, 0.6673471927642822, -1.0002862215042114, -1.6664574146270752, 0.8212490677833557, -0.505965530872345, -1.6372219324111938, 0.03453247994184494, 0.3469768762588501, 1.7118319272994995, -0.023825284093618393, 0.20545890927314758, -0.012990869581699371, 0.28209584951400757, 0.12511970102787018, -0.6263326406478882, -0.21179607510566711, -1.37358558177948, 0.7135441303253174, -1.2440295219421387], [-0.11801586300134659, -0.8986151218414307, 0.13748376071453094, 1.7350459098815918, 1.051168441772461, 1.7949429750442505, -2.5062689781188965, 0.7839872241020203, -0.07287043333053589, 1.3345773220062256, 2.134430408477783, -1.4852148294448853, 0.40806761384010315, 0.7398524284362793, 0.40179741382598877, 1.4427125453948975, 0.5312778949737549, -0.4136722683906555, -0.818430483341217, 0.7849225401878357, 0.3756565451622009, -0.06128670275211334, -0.2370179146528244, 2.5522067546844482, 1.165426254272461, -0.11423409730195999, -0.6234371662139893, 0.47325342893600464, -0.35389378666877747, 0.48241326212882996, 1.2216036319732666, 0.5239534378051758, -1.3030458688735962, -0.03586538881063461, 0.8251726031303406, -0.11685902625322342, 1.3927170038223267, 0.19795827567577362, 0.8289119005203247, 0.829756498336792, 0.34484535455703735, -0.40712568163871765, -0.1900065839290619, 2.9372849464416504, -0.2129269391298294, 0.21502520143985748, -0.23901286721229553, 0.5576980710029602, -0.17429547011852264, 0.7157043814659119, 0.6254705786705017, -1.3662699460983276, -0.856506884098053, -1.0426074266433716, -0.19058682024478912, 0.4789193272590637, 0.21401700377464294, -0.7259070873260498, -1.6279062032699585, -0.17832103371620178, -1.1810795068740845, -0.34451794624328613, -0.5559338331222534, 0.037198856472969055], [0.9185557961463928, -0.7545793056488037, -0.696209728717804, -1.3395963907241821, -0.6530110239982605, 0.14851334691047668, 1.4441457986831665, 0.34547096490859985, -0.13849279284477234, 0.28880149126052856, -0.5731000900268555, -1.2694272994995117, -0.23971021175384521, 1.802459716796875, 0.6997348666191101, 0.2443663328886032, -1.1008130311965942, -0.19520029425621033, 0.43540945649147034, -0.5252057313919067, -0.04636969044804573, -0.3094443380832672, -0.434546560049057, 0.4429677724838257, -0.09459362179040909, 1.9124531745910645, -0.7417489886283875, 1.8000659942626953, 2.301441192626953, -1.0895922183990479, 0.1416979283094406, -1.462597370147705, -0.8192311525344849, 0.47046032547950745, 1.449639081954956, -0.30916711688041687, 0.3639046549797058, 0.11775342375040054, -0.18149127066135406, 1.0462677478790283, -0.49077826738357544, 1.0203640460968018, 0.3786149024963379, -0.9463703036308289, 0.10576300323009491, -1.5430951118469238, 1.83415687084198, 0.4460543394088745, 0.6102580428123474, -1.2385326623916626, 0.289673775434494, 1.0195858478546143, 0.674694836139679, 0.1757768839597702, 0.3435940742492676, 1.1990169286727905, 1.1179368495941162, 1.0195324420928955, -0.04155796021223068, -2.216346263885498, 0.6656848192214966, 0.7151316404342651, -0.8777168989181519, 1.6038966178894043], [1.3304542303085327, -0.4110592007637024, 1.8297220468521118, 0.9875336289405823, -0.9397552013397217, 1.256276249885559, -0.20176400244235992, 0.5135547518730164, -0.1306012123823166, 0.4323584735393524, 1.0255568027496338, 0.18895621597766876, -0.11973809450864792, 0.12721869349479675, -2.394897222518921, -0.10936202853918076, -0.9632409811019897, -1.1504007577896118, 0.6155887246131897, 0.7304937839508057, -1.0691934823989868, -0.8408939838409424, 1.6095612049102783, -1.6179747581481934, 0.6870173811912537, 0.42845991253852844, -0.029335519298911095, -0.8988569378852844, -0.3175428509712219, 0.6083095669746399, 1.297876238822937, -1.2294124364852905, -0.7180873155593872, -0.9984021782875061, 0.15843288600444794, 0.6327974200248718, 0.07921268790960312, 0.47254809737205505, -0.12331680953502655, -1.1398766040802002, 0.6307461261749268, 0.06385860592126846, -0.8483800292015076, -0.2912568151950836, -0.9748938679695129, 0.8177744150161743, -1.1428172588348389, 1.2198715209960938, -0.3342089354991913, 2.2619283199310303, -0.49329039454460144, -0.2719581723213196, -1.6699507236480713, -0.09779727458953857, -1.8610669374465942, 0.841396152973175, -0.3265894949436188, -0.7666801810264587, -1.078891634941101, -0.6848440170288086, 1.075379490852356, 1.332817792892456, 0.36633309721946716, 0.3823191821575165], [-0.12300838530063629, 0.224834144115448, -1.8982923030853271, 1.9621734619140625, -0.3937340974807739, 0.3452620208263397, 0.3995848596096039, 2.10719895362854, -2.268996238708496, -0.4501105546951294, 0.284807950258255, 1.1793744564056396, -1.7128005027770996, -0.14139841496944427, 1.0691988468170166, -0.9136548042297363, 1.6211210489273071, 2.9497008323669434, 0.42244330048561096, -0.7401513457298279, -0.4573023021221161, -1.0897852182388306, 0.20319883525371552, -0.5131227970123291, -0.5651809573173523, 0.16002508997917175, -1.1450843811035156, 0.8228944540023804, -2.303299903869629, 1.4350918531417847, -0.4341951608657837, 0.3054848611354828, -0.520251452922821, -0.9532920718193054, -0.2607191503047943, -0.2707734704017639, -0.3043435513973236, 1.4571179151535034, -1.5550541877746582, -0.758037269115448, 0.7813674211502075, -1.108934998512268, 0.1691105216741562, -0.8725440502166748, -0.9326542019844055, 0.9573150277137756, 0.6220192313194275, -1.383424162864685, 0.5500858426094055, -1.809340000152588, -1.5832273960113525, -0.48768648505210876, 0.2618854343891144, 0.06565190106630325, 1.0745949745178223, 0.6930711269378662, -1.4711183309555054, -1.2993944883346558, -0.1869998425245285, -1.1129271984100342, 0.8359649777412415, -0.6068062782287598, 1.7849199771881104, -0.8682546615600586], [-0.15940524637699127, -1.9813446998596191, -1.9954915046691895, 0.44961974024772644, -1.8525807857513428, 0.09751913696527481, 3.3068501949310303, -2.119382858276367, -0.09086211770772934, 0.1494973599910736, -0.6228864789009094, 0.2302895039319992, -0.2735746502876282, 0.6504231691360474, 0.09480343759059906, -1.7096779346466064, 0.4729013442993164, -0.17179054021835327, -0.49274736642837524, -0.34973108768463135, -1.3389604091644287, 0.5400886535644531, -0.17588798701763153, 1.3778407573699951, 0.46119916439056396, 0.9024648070335388, -0.32740482687950134, -0.723935067653656, -0.4684963524341583, -0.4776831269264221, 1.3783687353134155, -0.31344589591026306, 0.7103496789932251, 0.5679283142089844, -0.3325292468070984, 1.2075520753860474, 1.1647523641586304, -0.8790982365608215, 0.48700129985809326, -0.6300367712974548, -0.07186023145914078, -0.09744766354560852, 0.36118876934051514, 1.0992499589920044, -0.6706145405769348, 0.43212568759918213, 1.8067570924758911, -1.0388237237930298, -0.4800967872142792, -0.7371000647544861, -0.04911207780241966, 2.1040284633636475, -0.08866332471370697, 1.1669092178344727, 1.2942240238189697, 0.14039920270442963, -2.2326910495758057, -0.1803087741136551, -1.5083913803100586, 0.09143541008234024, -0.06959328055381775, -1.3821240663528442, -0.31281810998916626, -1.415342092514038], [0.8696597814559937, 0.6999537944793701, -1.2050715684890747, -0.0995728075504303, 0.6789953112602234, -0.5993448495864868, -1.3394511938095093, 0.04408758133649826, 0.2806997299194336, 1.2877469062805176, 0.8610497117042542, -1.1269363164901733, -0.850858211517334, -1.6679189205169678, -0.6090714335441589, 1.1731600761413574, 0.5462421774864197, -0.18924717605113983, 0.011970236897468567, 0.09935116022825241, -0.2931287884712219, 0.07427673041820526, -0.31024879217147827, -0.735722541809082, 0.8079257607460022, 0.45895272493362427, 0.8252671957015991, -0.7389599680900574, -0.20039156079292297, -0.8618167042732239, 0.4945995509624481, -0.7338835000991821, 0.08134014159440994, 1.6117244958877563, 0.5553364753723145, -1.6547479629516602, 0.8922443985939026, -0.8388482928276062, -0.6437920331954956, 0.5763276219367981, 1.147233247756958, -0.5643520355224609, 0.1013081967830658, -0.4035104513168335, -0.2441779524087906, 0.9678008556365967, 0.8317166566848755, 0.015600966289639473, -0.38973191380500793, -1.186808705329895, -0.4968973994255066, -0.523981511592865, 0.8091177940368652, 0.13290636241436005, -0.4905199110507965, 1.328410267829895, 0.6209239363670349, -0.07676225900650024, -2.741231679916382, -0.07936526089906693, 2.397890329360962, 0.6505054235458374, 1.3320541381835938, -0.9511512517929077], [-0.8003880977630615, -0.3317934572696686, -1.6184967756271362, -0.9142786264419556, -1.1652058362960815, -0.14736104011535645, 0.31867268681526184, -1.10901939868927, 0.6789307594299316, 1.4180505275726318, -0.6710365414619446, -0.38918620347976685, -0.2299385815858841, 1.6816508769989014, 0.14319482445716858, -0.5465798377990723, 0.9287046194076538, -0.5022879242897034, -0.554426908493042, 2.1615681648254395, -1.4147515296936035, 0.1885661631822586, -0.8774624466896057, 1.5893175601959229, 1.2973005771636963, -2.7374331951141357, 0.7968410849571228, 1.0619919300079346, 2.9658122062683105, 0.32944703102111816, -1.2725415229797363, -1.0302152633666992, 0.13328208029270172, 0.2760055363178253, -0.07249000668525696, 1.6670925617218018, 1.2618004083633423, -1.0141063928604126, 0.8741719126701355, -0.27352553606033325, -1.538345217704773, 0.6617295145988464, 0.8601375818252563, -0.15385252237319946, 0.17869886755943298, -0.6258466839790344, 0.4855712652206421, -2.0681164264678955, -0.10109566897153854, 0.5467821955680847, 0.18403008580207825, 0.7206512689590454, -1.390324592590332, -0.9535618424415588, -0.25048428773880005, 0.45649489760398865, 0.5264614820480347, 1.491606593132019, -0.1839800924062729, -0.12203243374824524, 2.652698278427124, -0.46142083406448364, 1.3699760437011719, -1.124862790107727], [-1.060052752494812, 1.555696725845337, -1.3112461566925049, 0.6812979578971863, -2.154574155807495, -0.29118773341178894, 0.6563639640808105, -0.4672858715057373, -0.44075608253479004, 0.30465057492256165, -1.6143598556518555, -0.6756905913352966, 0.24383047223091125, 0.01447717472910881, -2.0933353900909424, 1.626101016998291, -1.7965939044952393, 0.9545726776123047, 0.053280990570783615, -0.13305722177028656, -0.07641544938087463, -1.276922583580017, -0.553239107131958, -0.9317603707313538, 0.5708211064338684, -1.841217041015625, -0.42128273844718933, -0.28957226872444153, -1.0477750301361084, 0.4469166696071625, 0.48140257596969604, 0.4917367994785309, -0.22115379571914673, 0.6069026589393616, -0.6254737377166748, -0.6313828229904175, -2.106208324432373, 0.4601290225982666, -0.4442858099937439, 0.3825053572654724, -0.3379426598548889, -0.366280198097229, -0.7245038151741028, -1.4305254220962524, 0.07802952080965042, 0.04393474757671356, 0.5501510500907898, -0.15360122919082642, -1.4457037448883057, 0.7217047810554504, 0.6333090662956238, 0.014284881763160229, 1.297309160232544, 0.9339704513549805, -0.6935896277427673, 2.2915146350860596, -0.8545480966567993, 0.7973074316978455, -0.2846620976924896, -1.2917771339416504, -1.542132019996643, 0.2497103214263916, -0.6837755441665649, 0.3004133701324463], [1.5705152750015259, -1.7453367710113525, -1.0539911985397339, 0.30486541986465454, -0.6181333661079407, -0.9907760620117188, -1.3309085369110107, 0.07725966721773148, -0.6829038858413696, 1.820192813873291, 0.4922601580619812, -0.5299413204193115, -0.5799499750137329, 1.5477333068847656, -0.3647201359272003, 0.49120384454727173, 0.5969016551971436, 1.535207986831665, 0.7351087927818298, 1.144511342048645, 1.4251508712768555, 0.01812690682709217, 1.283090591430664, -1.811086654663086, 1.1320286989212036, -0.1857152134180069, -0.7126457691192627, 1.0314667224884033, 1.2116280794143677, 1.630445957183838, 0.17770907282829285, 0.4244197905063629, 0.6266494393348694, -0.11444321274757385, 0.021448401734232903, -1.9276155233383179, -0.24991996586322784, 0.7053564786911011, 1.5481590032577515, -1.5004485845565796, -0.41675588488578796, -0.9187684059143066, -0.029791593551635742, 0.907404363155365, -1.3666516542434692, 0.2865404486656189, -0.04833242669701576, 1.2013418674468994, -1.4128071069717407, -0.5286474227905273, 1.1112074851989746, -0.06450449675321579, 1.549215316772461, 0.9409735798835754, 1.2971569299697876, -0.2401774525642395, 0.5938422679901123, 0.123694509267807, 0.8413960933685303, 0.7268744707107544, 0.1831788420677185, -0.4885353147983551, 1.1921498775482178, 0.3543632924556732], [0.5845468640327454, -1.070145606994629, -0.5588778257369995, -0.06601941585540771, -0.5278734564781189, -1.3646950721740723, 0.2481222152709961, -0.4349169135093689, -0.08128707110881805, -0.3910481929779053, -1.106396198272705, 1.3997634649276733, -1.958658218383789, -1.4366804361343384, -0.4292130172252655, -1.0069735050201416, -1.409830093383789, 1.224186658859253, -1.2920284271240234, -0.32912272214889526, -0.6095876693725586, 1.1222878694534302, -0.08080362528562546, 0.21036645770072937, 1.2732653617858887, 1.1500535011291504, 1.432658076286316, 0.038366589695215225, 0.2218104898929596, -1.2572327852249146, -0.7667787671089172, 1.4088091850280762, 0.9268565773963928, -1.4253768920898438, 1.4136377573013306, 0.25959765911102295, -0.3774036765098572, 0.06550966203212738, 1.0783685445785522, -1.9737035036087036, 0.6347567439079285, 0.7603267431259155, 1.267699122428894, 0.2199086993932724, -1.4357531070709229, -1.2803009748458862, -1.4692473411560059, -1.0076870918273926, 1.2987388372421265, 0.24336287379264832, 0.8277409672737122, 0.37652602791786194, 0.4928857982158661, -0.3137952387332916, -2.5176355838775635, -0.289084792137146, -0.36052101850509644, 1.1551127433776855, 1.4681472778320312, 1.4758445024490356, -1.1452350616455078, -0.22505301237106323, 0.1027972549200058, 0.4988393187522888], [0.30983662605285645, 0.41917356848716736, 0.4278992712497711, -1.7971400022506714, -0.9915343523025513, -1.425542950630188, 1.7354393005371094, -0.8391237854957581, 0.7723971605300903, 0.4463081359863281, 0.1898353099822998, 1.3222779035568237, -0.7845903635025024, 1.0396391153335571, -0.7213855385780334, -0.5187221169471741, -0.13797330856323242, -1.559161901473999, 0.24248424172401428, 1.4713413715362549, 1.1549900770187378, -0.9491047859191895, 2.0907163619995117, 2.1604249477386475, 0.42248469591140747, -0.19196666777133942, 0.6183972358703613, 1.2695724964141846, -0.13548947870731354, 0.9735463857650757, 2.1284823417663574, -0.6254279613494873, -0.8723077774047852, 0.9664689898490906, -0.09385120123624802, 0.031165139749646187, -0.5648324489593506, 0.8157956004142761, 0.31514212489128113, -0.22521959245204926, -0.7351458072662354, -0.04162780940532684, -0.4455690085887909, 1.0371533632278442, -0.22908134758472443, 0.8248011469841003, 0.9831915497779846, -0.018254203721880913, 0.4577194154262543, -0.04494102671742439, 1.3354257345199585, -0.602918803691864, 1.1011524200439453, -0.011943885125219822, -0.11717166006565094, 0.27242162823677063, -0.5817764401435852, 0.19520220160484314, -0.21218694746494293, 0.05148966982960701, 1.1478785276412964, -2.1150481700897217, 1.009723424911499, 0.5734614729881287], [-0.32798516750335693, 0.17447608709335327, 0.1723833531141281, -0.04405008256435394, 1.6602307558059692, 0.6325535774230957, -0.39186206459999084, 0.25437483191490173, -0.10880573838949203, -0.1374533474445343, -0.19431769847869873, -0.5539108514785767, -0.23049475252628326, 0.10748393088579178, 1.7232208251953125, 1.2291014194488525, -0.8795713186264038, -1.086168646812439, -1.5683085918426514, -0.6332255601882935, -0.13969384133815765, 1.5686577558517456, -1.201522946357727, 0.8173389434814453, 0.8317791223526001, -1.5766024589538574, 0.6214467883110046, -0.48703819513320923, 0.4426998496055603, -0.15662822127342224, -0.8162402510643005, -0.37075838446617126, -1.7296340465545654, -0.2750623822212219, 1.1641347408294678, 0.674595296382904, -0.8833741545677185, -0.46048393845558167, 0.9118451476097107, -0.6552942395210266, 0.028385760262608528, 0.25544291734695435, -1.758305549621582, -0.013465489260852337, 1.2070385217666626, -0.18329057097434998, 0.7716721892356873, 0.20015199482440948, 1.0803680419921875, -0.6150851845741272, 1.4988503456115723, 0.05171719193458557, -1.186349868774414, -0.45981693267822266, -0.3608167767524719, 0.3898272216320038, 0.6682129502296448, 0.000568379124160856, -0.77342289686203, 0.7869521975517273, 0.6135355830192566, 0.32733142375946045, -0.26463082432746887, 0.6365107297897339], [0.32432273030281067, 0.28334832191467285, -0.11177857965230942, -0.9278178811073303, 0.11062972247600555, -0.07029309123754501, 0.5510737895965576, -0.09447483718395233, 0.9718414545059204, 0.23809142410755157, 1.4180957078933716, 1.4777631759643555, 2.898545503616333, 1.402276873588562, -0.5646704435348511, -0.7948271036148071, 0.5104066133499146, -0.5777268409729004, 0.7634589672088623, 0.6402475237846375, 0.5025970339775085, -0.9859760403633118, 0.0912899300456047, 0.710258960723877, 0.029000408947467804, 0.06653283536434174, -0.7412163615226746, 0.9789551496505737, -0.7329639196395874, -1.1385806798934937, 0.546184241771698, 0.1119188591837883, 0.24614766240119934, 1.460331916809082, -0.06236857548356056, -0.4998154640197754, 1.25151526927948, -1.2218072414398193, 2.1617050170898438, 0.9090713262557983, 1.0377812385559082, -0.5944300293922424, 1.9267408847808838, -0.2389211654663086, -1.1473536491394043, 0.06759383529424667, 0.3055032789707184, -0.377034455537796, 1.3094031810760498, -0.4409327805042267, 1.3884624242782593, -0.4364359378814697, 0.33243006467819214, 0.7805100679397583, -1.2750660181045532, -0.7898587584495544, -0.7022247314453125, 0.6276817917823792, 0.0008753747097216547, 1.1948895454406738, 0.19047211110591888, -1.2488411664962769, 1.6880335807800293, -0.08127453923225403], [-0.5322255492210388, 1.8852829933166504, -0.3698236048221588, 0.11205706000328064, 1.1435989141464233, 1.1159071922302246, 0.6177042722702026, -1.7892298698425293, -1.0192418098449707, 1.4364372491836548, -0.5990322828292847, -0.40256693959236145, -1.252524733543396, 0.9948364496231079, 1.2701658010482788, 1.4237353801727295, 0.10351788252592087, 1.207819938659668, -0.771239697933197, -0.6637914180755615, -2.513969659805298, 0.4664476811885834, 1.015034794807434, -0.05458603799343109, 0.3090170621871948, 1.9493403434753418, 0.08027691394090652, -0.15058936178684235, 0.13967782258987427, -1.6293754577636719, -2.068277597427368, 0.435683012008667, -0.3066352307796478, 2.044970989227295, -0.0791095569729805, 0.5068434476852417, -2.6686081886291504, -1.841732382774353, -0.976359486579895, -0.09890072047710419, -0.7638813257217407, 0.4580644965171814, 1.389328956604004, 1.1316697597503662, 0.2115224152803421, 0.2938399016857147, -0.7298539280891418, -0.7574230432510376, 0.13178306818008423, -1.0396510362625122, -0.09152081608772278, 0.1529165804386139, 0.6030694246292114, -0.6884766221046448, 0.3696361482143402, -0.9094405770301819, -0.03215968981385231, 0.43203890323638916, 0.8930795788764954, 0.5513088703155518, 0.9447181820869446, -0.6258333325386047, 1.3619123697280884, -0.5689296722412109], [-0.41584593057632446, -0.12634587287902832, 0.28709158301353455, 0.07827853411436081, -0.6138699650764465, -0.18813931941986084, -0.10125580430030823, -0.9099892377853394, -1.3196080923080444, 1.1297332048416138, -1.187665343284607, 1.6480735540390015, 0.5472939610481262, 1.1553962230682373, 0.03842930495738983, 2.3254284858703613, -0.108791284263134, -1.2300363779067993, -0.9199801683425903, 0.018672490492463112, -0.7776791453361511, 0.002444707090035081, 1.328634262084961, 0.3653430640697479, -0.3780851662158966, -1.034217119216919, -1.0582648515701294, 0.5651684999465942, -0.4776989817619324, -0.6109466552734375, 1.6873518228530884, -0.9916708469390869, -1.0534119606018066, -0.7187228798866272, 0.4390122890472412, 0.3201707899570465, -0.3707893490791321, 0.9319235682487488, 0.810577929019928, 0.9940834641456604, 1.106367588043213, -0.03055371530354023, -0.47890299558639526, 0.04724182188510895, 0.430753618478775, -0.6056087613105774, -0.4829231798648834, 0.5541185140609741, -2.1484930515289307, -0.5246391296386719, 0.22642512619495392, -1.2419384717941284, -0.4121758043766022, 0.3774106502532959, -2.5913772583007812, -0.6626132726669312, -0.4440601170063019, -0.8014391660690308, 2.0571045875549316, 1.8244186639785767, -0.31793832778930664, -0.9119633436203003, -0.36104920506477356, 0.02798858843743801], [-0.18914413452148438, 0.3872941732406616, 0.30229902267456055, -2.481323480606079, -1.219104528427124, -0.018703967332839966, -0.36425116658210754, 0.44029703736305237, -0.5442154407501221, 0.8771390318870544, 1.847705364227295, -0.08949358016252518, 0.6426504254341125, -0.3782689869403839, 1.1086952686309814, 0.9177053570747375, -0.03559296950697899, 0.4318854510784149, -0.5228484272956848, -1.3705676794052124, -2.667631149291992, -1.4534937143325806, -2.166642427444458, 0.8026446104049683, -0.35622113943099976, -1.0352133512496948, -1.2625415325164795, -0.7597704529762268, -1.4785770177841187, -0.06345853209495544, 1.210580587387085, 1.0484836101531982, -0.14032091200351715, 0.22022777795791626, -0.9618010520935059, -0.8928980827331543, 0.6925877928733826, -1.4629309177398682, 1.423296332359314, 0.46812984347343445, -0.0026180888526141644, 1.6859720945358276, 0.8626099824905396, 1.1716872453689575, 0.148900106549263, -0.4718843102455139, -1.9256328344345093, -1.4701741933822632, -0.07899155467748642, 0.6753581166267395, 1.2720937728881836, 0.28477075695991516, 3.0185298919677734, 0.8698201179504395, 0.03192885220050812, 1.9118849039077759, -0.7714917063713074, 1.383009672164917, 1.2868058681488037, 1.1232075691223145, 1.793262243270874, -0.38643917441368103, 1.1479156017303467, 1.1846445798873901], [-0.7774812579154968, 0.009266957640647888, 0.6565184593200684, -0.472466379404068, -2.1972029209136963, 0.5078871846199036, -2.359097957611084, -0.01351255364716053, 2.262683391571045, -1.7345290184020996, 1.656802773475647, 1.133376955986023, 1.0523496866226196, 1.188489317893982, 2.0706019401550293, 1.3940787315368652, -0.8205703496932983, 0.4532430171966553, -1.1282285451889038, -0.8189302682876587, 0.37846583127975464, 0.32878586649894714, 1.0077824592590332, -1.395055890083313, 0.5934546589851379, -0.8302021026611328, 0.8149105310440063, -0.5493912696838379, -0.34489700198173523, 0.5527864694595337, 0.6421026587486267, -0.19572243094444275, -0.15039998292922974, -0.10317473858594894, -0.7994712591171265, -0.5346753597259521, -0.31500086188316345, 1.4198955297470093, 0.1488773077726364, 0.5496103167533875, -0.4956471621990204, -0.5104082226753235, -0.38349464535713196, -1.5738425254821777, -2.3443284034729004, 0.581399142742157, 1.3841273784637451, -0.43604162335395813, -0.18147531151771545, 0.1650635302066803, 0.22163742780685425, -0.5784803032875061, 1.356554627418518, -1.8511247634887695, 0.14097987115383148, 0.555669367313385, -1.464536190032959, -1.9309343099594116, -0.481229305267334, 0.43111300468444824, 0.5562770962715149, -0.368767648935318, -1.079530119895935, -1.108137845993042], [-1.0053601264953613, -1.5147314071655273, -1.8743666410446167, -0.49977728724479675, 0.7914795279502869, -0.3182452619075775, -0.09081908315420151, 1.0511313676834106, -0.9591147303581238, 0.7935296297073364, -0.2897382378578186, 1.7276785373687744, -0.19462841749191284, 0.5624751448631287, 0.011364671401679516, -0.20577943325042725, 0.987209141254425, 0.6565426588058472, 0.4904458522796631, 0.1554035246372223, -2.8814125061035156, -1.877376675605774, 1.629523515701294, -1.053765058517456, 0.7852082848548889, -0.1646147519350052, 1.24341881275177, 1.8177881240844727, 1.440529227256775, 0.6876838803291321, 0.46167218685150146, 0.7962209582328796, 0.09828279912471771, -0.7292215824127197, -1.6013208627700806, 1.4171321392059326, -1.296190619468689, 0.47179314494132996, 0.32504644989967346, 0.08893539756536484, -0.0303272046148777, 0.026232462376356125, -0.5283483862876892, 0.4412752091884613, 0.6696909666061401, 1.0723272562026978, 0.6000011563301086, 0.47739455103874207, 0.5550497770309448, -0.04938874766230583, -0.5564268231391907, -2.30367374420166, 0.6195091605186462, 0.8932200074195862, -1.5018401145935059, -0.6843582987785339, 0.9457778930664062, -0.0652046874165535, 2.088717460632324, -0.5592922568321228, -0.11478936672210693, -0.8243037462234497, 1.4486427307128906, 0.4760299026966095], [0.30467382073402405, -1.0726630687713623, 2.0064945220947266, 0.935784101486206, -1.3190349340438843, 0.1987946480512619, -0.5440477132797241, -0.82686847448349, 0.31995800137519836, 0.26141729950904846, -1.0515086650848389, 0.10689735412597656, -0.46323147416114807, 0.502190113067627, -0.9338497519493103, -0.09985227882862091, -0.05785254389047623, -0.21740147471427917, -1.2226487398147583, 0.8309996128082275, 1.0809110403060913, -1.460649847984314, 0.5378007888793945, -0.5024597644805908, -0.748530387878418, -0.48352181911468506, 1.494552493095398, -0.31021246314048767, -1.0768855810165405, 0.6851491928100586, -0.5779203772544861, 1.2243868112564087, 0.836847186088562, -0.930902361869812, -0.3658657371997833, 0.4289816915988922, -0.10084450989961624, 0.284823477268219, -1.1148059368133545, 0.06331847608089447, 0.8792058229446411, -0.6908899545669556, -1.400700330734253, -0.1241052970290184, -1.0153900384902954, 0.46107396483421326, 0.8221209645271301, 0.56839919090271, 0.5631588697433472, -0.32055437564849854, -0.4377061128616333, -1.2161225080490112, -0.19902098178863525, 1.7630493640899658, 0.816020667552948, -0.6579605340957642, 0.025546979159116745, 0.07955212891101837, -0.47057458758354187, 2.1546852588653564, -0.496687650680542, -0.2966093122959137, -1.9703280925750732, -1.5906387567520142], [-1.417025089263916, 0.05702463164925575, 0.20836803317070007, -0.4347614049911499, -0.2607453465461731, 0.4800140857696533, 0.7730123996734619, -0.9969733357429504, -0.6196221113204956, 0.3120099604129791, -1.12921142578125, 0.4533859193325043, -1.7467975616455078, -1.725512981414795, -0.6608782410621643, 1.0139567852020264, -2.6375372409820557, 1.0102380514144897, -2.4010908603668213, 0.06465552747249603, -0.2525804042816162, 0.8024436235427856, 1.9303942918777466, -1.3182094097137451, 0.025826187804341316, -1.3649725914001465, -0.3800818920135498, -0.3359247148036957, -1.5637295246124268, 1.1384563446044922, 0.22419027984142303, -1.1138454675674438, 0.44956621527671814, -1.0115971565246582, -0.443739652633667, 0.5068971514701843, -0.30562925338745117, -0.2549227476119995, -0.16374240815639496, -0.2298671156167984, 0.22165092825889587, -0.057692091912031174, -0.224024698138237, 0.5463066697120667, 0.5281969308853149, -0.36294442415237427, 0.6585227251052856, -1.141605257987976, -1.1109098196029663, -2.646043062210083, -0.19512678682804108, -0.7247924208641052, 0.01975756511092186, -0.21567651629447937, 1.7708879709243774, -0.5977388620376587, -0.8096629977226257, 1.7317811250686646, 2.1661410331726074, 0.8994251489639282, -0.6168360710144043, -0.3610413670539856, 0.49991652369499207, -0.6053112745285034], [-1.2212064266204834, 0.15919280052185059, -1.5488322973251343, -1.0153462886810303, 0.5576695799827576, -1.1478439569473267, -0.4366435408592224, 1.1490367650985718, 0.494460791349411, -0.18166327476501465, 2.014376640319824, 1.616564393043518, -0.5193851590156555, 0.1542312055826187, 0.9657164216041565, 0.24826563894748688, 1.1509448289871216, 0.23181524872779846, -1.1009153127670288, -1.0750651359558105, -0.931096076965332, -0.5166254043579102, 0.6759282946586609, -0.7231035232543945, 0.5437906384468079, -0.19552308320999146, 0.2632678151130676, 0.992290198802948, 0.4127122759819031, 0.9189260601997375, -0.4223857820034027, 2.0410566329956055, 0.8862094879150391, -1.6610162258148193, -0.37501591444015503, -0.03663141652941704, 2.132725954055786, -1.1566427946090698, -0.4569348990917206, -0.45619356632232666, -0.26785656809806824, 0.5360447764396667, -1.1758756637573242, 0.9830663204193115, 0.5748653411865234, -1.1870191097259521, -0.5916796326637268, -1.2678718566894531, 1.4264341592788696, -0.689941942691803, 1.1156113147735596, -1.8189988136291504, 1.1062251329421997, -1.4048008918762207, 1.5964447259902954, -1.3627153635025024, 1.662399172782898, 2.6751835346221924, 0.8033148050308228, -0.6538629531860352, 1.1950783729553223, 0.3365824222564697, 0.21800024807453156, -1.1120611429214478], [0.9799924492835999, -0.4958863854408264, 0.37822234630584717, -0.6655119061470032, -1.3790037631988525, -0.1715281754732132, 1.464255452156067, -1.0657658576965332, -0.5784721374511719, -0.9741350412368774, 0.33517932891845703, -0.5034324526786804, 0.8424601554870605, 0.1282988041639328, -1.2601481676101685, -0.29148855805397034, 1.0463862419128418, 0.49475857615470886, -2.3514270782470703, -3.0189263820648193, -0.4111068546772003, -1.8450679779052734, 0.060853783041238785, 0.7054387331008911, 0.9497776627540588, 1.4259564876556396, 0.8760947585105896, 0.43763551115989685, -0.5847562551498413, 0.4765614867210388, -0.4072185158729553, 0.13865533471107483, -1.1790165901184082, 0.06115777790546417, 0.3351787030696869, 1.374925136566162, 1.1342298984527588, -1.2740751504898071, 0.07955396920442581, 0.09683831036090851, 1.0153294801712036, 0.5973973274230957, -1.7120792865753174, -0.4437185227870941, -0.25830286741256714, -1.374536395072937, -0.3882392942905426, -0.5091332793235779, 3.3188400268554688, 0.7777546048164368, 1.6170152425765991, 0.26994219422340393, -1.452465295791626, -0.9492824673652649, 0.5882865786552429, 0.023513343185186386, 0.3482087552547455, 0.70451420545578, -0.13318486511707306, -3.0672554969787598, 1.2914057970046997, -1.5529541969299316, 0.41155996918678284, 0.5768767595291138], [-0.4497533142566681, 0.444870263338089, -0.2740003764629364, -1.7216132879257202, 1.6656267642974854, 0.8460418581962585, 0.42288121581077576, 1.0153439044952393, -2.2566018104553223, 1.1454894542694092, -0.09996159374713898, 0.9605496525764465, 1.2022876739501953, 0.7484390139579773, 0.18533459305763245, -1.4480019807815552, -0.35528603196144104, -0.3037765622138977, -0.6048937439918518, -0.9574642777442932, -1.046566367149353, 0.21360749006271362, 1.1839699745178223, 0.24831470847129822, 0.43479764461517334, -0.8512882590293884, -0.9859766364097595, -0.6430421471595764, -0.7040305733680725, -0.4949544370174408, -0.131757915019989, 1.1036102771759033, -1.3954851627349854, -0.6187820434570312, -0.009722422808408737, 0.7110231518745422, 0.585239827632904, 0.01693321391940117, 0.35482150316238403, -0.447623074054718, 0.8621091246604919, -0.0008546048193238676, 0.2400549203157425, 1.1007038354873657, 1.8134040832519531, -0.11447600275278091, 1.586397647857666, -1.3502715826034546, -0.9333199262619019, -1.632071852684021, -1.699663519859314, 0.6340996623039246, -0.20748871564865112, -0.11673618108034134, 0.6602206826210022, 0.7709857225418091, 1.1994906663894653, 0.16312351822853088, -0.5314438939094543, -0.9462043046951294, 0.21081361174583435, 0.30834105610847473, -0.3057810366153717, -0.3918296694755554], [-0.32603198289871216, -0.7556399703025818, -0.7091147303581238, 0.7274092435836792, 1.016150951385498, -0.704460620880127, -0.22386395931243896, 0.683977484703064, 0.1015324667096138, -0.4605141580104828, -0.3327235281467438, 0.17510992288589478, 1.4447875022888184, 0.7194088101387024, 1.779761791229248, -0.31486132740974426, 0.4648967385292053, 1.181070327758789, 0.07005620747804642, -0.9175134301185608, -0.27160266041755676, -0.30700528621673584, -0.029436171054840088, 0.46080517768859863, -0.3072904944419861, 0.11754953861236572, 2.6485164165496826, -2.145782232284546, 0.08643398433923721, 2.277545928955078, 0.3597395122051239, -0.024367982521653175, -0.5748584866523743, 1.069892406463623, 0.8130706548690796, 1.1115370988845825, 1.995442271232605, -1.1231054067611694, 2.4840590953826904, 0.3981611132621765, -0.9762730002403259, 0.2760613262653351, -1.29482901096344, -0.36007341742515564, 0.3438120186328888, -0.9828087091445923, 0.8148163557052612, 0.15247249603271484, -1.1740165948867798, -0.29876208305358887, 0.4058040380477905, 1.8291152715682983, -1.1158548593521118, -1.4072681665420532, -2.5606842041015625, -0.0623384490609169, 0.27267175912857056, 0.5854012966156006, 0.4615795910358429, -1.4278333187103271, 0.42522138357162476, -0.7141873240470886, 0.3382986783981323, -0.08168131113052368], [0.37655794620513916, 0.4906310737133026, 0.8258360028266907, 0.2997289001941681, -1.5723179578781128, -0.17559467256069183, -0.5410350561141968, 0.5611934661865234, -0.08579376339912415, -0.6924339532852173, -0.8547563552856445, -0.3807884156703949, -0.2914546728134155, 0.9894248247146606, -1.3800476789474487, 2.1262001991271973, -1.000378131866455, -0.03365347906947136, -0.1086224764585495, -0.05123685300350189, 1.4118502140045166, 0.3212157189846039, -0.06297912448644638, 0.39095547795295715, -2.3295252323150635, -1.3345032930374146, 0.3901626169681549, -0.6282296776771545, -0.8407254815101624, 0.8813083171844482, 0.3604535460472107, 1.2437034845352173, 0.9219152331352234, -0.1777900904417038, -1.2587369680404663, -1.4339523315429688, 0.19671942293643951, -1.8832005262374878, -0.5499173998832703, -0.07418496161699295, 1.6882027387619019, -2.867398500442505, -1.5460604429244995, -0.4819849729537964, 0.27646782994270325, 0.28573355078697205, -0.9772489666938782, 1.4522924423217773, 0.7293654680252075, 0.6674820184707642, 0.07100702822208405, 1.6001181602478027, -0.7123311758041382, -0.2921406924724579, -1.4825445413589478, -1.70671808719635, -0.04280160740017891, 1.2389177083969116, -0.8611959218978882, -0.09522926807403564, 0.1970638781785965, -1.7570306062698364, 0.7900282740592957, 1.0221015214920044], [-1.3662819862365723, -0.9989255666732788, 0.6788535714149475, 1.2128918170928955, -1.1405572891235352, 0.9240337610244751, -1.593239665031433, -1.002994179725647, 0.6687273383140564, 0.10573142021894455, 2.047442674636841, -0.9563993215560913, 1.461759328842163, 0.47717946767807007, 0.007276354357600212, -0.33493590354919434, 1.0417149066925049, 1.2260466814041138, -0.9056535363197327, -0.23599596321582794, 0.7102179527282715, 0.5249938368797302, -0.1273852288722992, 1.5022464990615845, 2.188560962677002, -1.5925679206848145, 1.3010995388031006, 0.8104450702667236, -0.9446479082107544, 0.844787061214447, -1.1899566650390625, 0.349025696516037, -0.1772112101316452, 0.1276826709508896, 0.10599717497825623, 1.1935094594955444, 1.2171627283096313, -0.3721577227115631, 0.09474754333496094, 1.6457849740982056, -0.8046848773956299, -0.07472271472215652, 0.3508927524089813, 0.36557552218437195, -1.5990349054336548, 1.527422308921814, 0.7568917274475098, -0.48258891701698303, 0.9907031655311584, 1.1529686450958252, 1.875719428062439, -0.6448817849159241, -1.0833085775375366, -0.8169780969619751, -0.13318566977977753, 0.6626085042953491, -0.08675850182771683, -0.5022238492965698, 1.1294374465942383, 0.5929276347160339, 0.05404777079820633, -0.2070280760526657, 1.5344245433807373, -1.7765589952468872], [-0.7938320636749268, 0.7687753438949585, -0.042594026774168015, -1.4724438190460205, 0.48244190216064453, -0.3052524924278259, 1.2150312662124634, -0.8452672958374023, -1.2830345630645752, -0.06596563756465912, -0.6638090014457703, -0.7755250930786133, -0.31111428141593933, 2.039170742034912, -0.3398803472518921, 2.2325665950775146, -0.19029386341571808, 1.951230764389038, 0.5389966368675232, 0.4407486319541931, -0.9886800646781921, 0.0022369108628481627, -0.9699978828430176, -0.13692989945411682, 0.15816304087638855, 1.6675199270248413, 0.08789084851741791, 0.9067310690879822, 1.9820098876953125, -0.9343395829200745, 0.4454137086868286, 0.9051420092582703, -0.779346764087677, 0.20592772960662842, 0.33024704456329346, -1.8265913724899292, -0.01142246462404728, 0.02362680993974209, -0.5311262011528015, 2.253514051437378, 2.282991409301758, 0.14034008979797363, 0.3660501539707184, 0.6414306163787842, 2.6051559448242188, 0.5631902813911438, 0.9953808188438416, -1.0693269968032837, -0.08814054727554321, -0.7329561710357666, 0.46638694405555725, 1.5838543176651, -1.5452054738998413, 0.6448811292648315, 1.015559434890747, 0.9999095797538757, -1.1522150039672852, 0.5716645121574402, 0.546108603477478, -1.1776188611984253, -0.7293141484260559, 0.26336172223091125, 2.9261417388916016, -1.3243157863616943], [0.31388312578201294, -0.2925492525100708, -0.9313449859619141, -0.37081965804100037, -0.3643464744091034, 2.3292040824890137, -1.5603846311569214, 0.19914644956588745, -1.1268377304077148, -0.4240633547306061, 0.1709323227405548, 1.3497889041900635, 0.8952726721763611, -1.0558946132659912, -0.8432636857032776, -2.5562849044799805, -0.027233608067035675, -0.47111597657203674, -1.6100691556930542, 0.9569649696350098, -0.16083991527557373, 0.11771439760923386, -0.1419925093650818, 0.5993730425834656, -0.9485893249511719, -1.5695395469665527, 0.8150103092193604, -0.003470424562692642, -0.1211959645152092, -1.097084403038025, 0.2778327763080597, -1.1022915840148926, 0.8103114366531372, -2.744086742401123, -2.8784098625183105, 0.04908332973718643, 0.963504433631897, -1.470330834388733, 1.4650001525878906, 0.2055441290140152, 1.323265790939331, 0.4604223072528839, -0.333805114030838, -0.3574346601963043, 0.2712056636810303, 0.16904959082603455, 0.7069088816642761, 0.3581269085407257, -1.3628296852111816, 1.150688648223877, 0.5989601016044617, -0.7916432619094849, -1.1765260696411133, 0.7384061217308044, 2.2130236625671387, -1.5277072191238403, 0.8712961077690125, -1.246915578842163, 0.10733281821012497, 0.8078671097755432, 0.06265730410814285, -0.5994869470596313, 1.124053955078125, 0.40114012360572815], [0.3012787103652954, 1.8351093530654907, -1.2759380340576172, -0.7623128294944763, 0.9455976486206055, -1.5573537349700928, 0.9043187499046326, -0.7364907264709473, 0.9448044300079346, -1.2921706438064575, -0.15500310063362122, -0.1643933802843094, -0.9277225136756897, 0.07532612234354019, -0.4562205672264099, 0.42220765352249146, 1.2557164430618286, -1.8108922243118286, 1.0241260528564453, 0.6069759726524353, -0.2957803010940552, 1.0855711698532104, 0.6547724604606628, 0.5869686007499695, 1.2416237592697144, 0.2774081230163574, -1.1540058851242065, -0.2480669915676117, -1.214248538017273, 0.0041933064348995686, -2.169276475906372, -0.6040189862251282, 1.173557162284851, 1.1813024282455444, 0.09655609726905823, -0.1688384860754013, -0.6979682445526123, -0.5557438731193542, -1.1257528066635132, -1.9766854047775269, 0.3412078619003296, 1.4923548698425293, 0.15119679272174835, 2.133878231048584, 0.24131329357624054, 0.2781389355659485, -1.510572910308838, 1.5584076642990112, -0.6201333403587341, 0.8226834535598755, 1.106361746788025, 0.43551763892173767, 0.23223020136356354, -0.20500698685646057, 0.4424969255924225, -0.043660860508680344, -0.2147170752286911, -1.5764868259429932, 0.7571077346801758, 0.9560022354125977, -1.9989261627197266, -1.0108051300048828, -0.8631622195243835, -0.24297164380550385], [0.46901440620422363, -0.7689433693885803, 0.14337219297885895, -1.538989543914795, 1.195738434791565, -1.4784108400344849, 2.066812038421631, 0.14250214397907257, 1.9245470762252808, 0.9923158288002014, -0.23619066178798676, 1.3485004901885986, -1.3114770650863647, -1.1533019542694092, 0.6795544624328613, 0.4597489833831787, 1.3213194608688354, -2.1143386363983154, -0.9640623331069946, -0.16493554413318634, 1.6982547044754028, 0.5072323679924011, 0.1731659173965454, -0.44414016604423523, -0.8902367353439331, -0.16835953295230865, -0.8165057301521301, -1.2322667837142944, -0.5354223251342773, -1.5959820747375488, 0.40180185437202454, -0.6773492693901062, -0.165359765291214, 0.87523353099823, -0.5737218260765076, 1.0361357927322388, -0.7863863706588745, -0.6468569040298462, 1.1102268695831299, -0.03299888223409653, 0.08980348706245422, 0.16105596721172333, 1.1396698951721191, -0.3583620488643646, -0.8648039698600769, 0.8396790623664856, 0.6240670680999756, -0.6038883924484253, 1.0600595474243164, 0.5320234298706055, -0.7649329900741577, 0.8579169511795044, 0.6349346041679382, -2.155000925064087, -1.5287137031555176, 0.7681331634521484, -0.8139715790748596, -0.3938537538051605, -1.0945780277252197, -1.4281399250030518, 0.6835399866104126, 0.31448233127593994, 0.33693167567253113, 0.19189657270908356], [-1.2810357809066772, 0.49243849515914917, 1.9136844873428345, -1.1279747486114502, 0.31500038504600525, 0.3615586459636688, -1.9289507865905762, 1.827147364616394, 0.08788303285837173, -0.12795865535736084, -1.0472041368484497, -1.3255723714828491, -0.11222890019416809, 1.8154842853546143, -1.3293187618255615, -0.2542566955089569, 0.2517651319503784, 0.846362292766571, 0.2672896981239319, 0.4326380789279938, 1.561787724494934, 0.20379498600959778, -2.102072238922119, -0.737178385257721, -1.2010473012924194, -0.9193260073661804, 0.16681252419948578, -0.7113348841667175, 0.8929615020751953, 0.32874926924705505, -1.160174012184143, -0.718578040599823, -1.0316139459609985, 0.9652486443519592, -0.11362382024526596, -0.7946875095367432, 2.0685694217681885, 0.034000739455223083, -0.8616864681243896, 1.3473281860351562, -0.03972625732421875, -0.39068445563316345, 0.1052989587187767, 1.011818766593933, 0.9254353046417236, -0.17880742251873016, 0.01814626157283783, 1.1813321113586426, 0.5345130562782288, 1.2992459535598755, 0.44056767225265503, -0.7775516510009766, 0.41966015100479126, -0.635913074016571, 0.44860029220581055, 0.09152870625257492, 0.33289942145347595, 1.3698067665100098, -0.5868739485740662, 0.3317018747329712, -0.022102180868387222, 0.07286927103996277, -1.9013776779174805, 0.4619012773036957], [-0.008641323074698448, 1.088577151298523, 1.1696081161499023, 1.0610318183898926, -0.5450652837753296, -0.6189746260643005, -0.3963046967983246, 0.028104033321142197, -0.7854832410812378, 0.6594797372817993, -0.5319099426269531, -1.1489747762680054, 0.028853360563516617, -0.4739567041397095, 0.6959465146064758, 0.6713840365409851, 1.038527488708496, -1.7270997762680054, 3.493190288543701, 0.0118740014731884, 0.7265536785125732, -1.2361209392547607, 0.13367925584316254, -0.931195080280304, 0.08872523903846741, -0.7442852258682251, -2.5528922080993652, -0.001666341326199472, -0.7617687582969666, -0.5259289145469666, 1.4513732194900513, 0.08033302426338196, -0.06327664107084274, 0.06615600734949112, -0.9776147603988647, -0.6880953311920166, -0.013437474146485329, 0.17691108584403992, -0.5601816773414612, 0.016253191977739334, 2.0096163749694824, -1.8883894681930542, 0.6914041042327881, 0.25963491201400757, 1.287817120552063, -0.42926010489463806, 0.9028151631355286, 1.3208409547805786, -0.30893006920814514, -0.7792514562606812, -0.04750499874353409, -0.13351130485534668, 1.303468108177185, -2.3628926277160645, 0.6057058572769165, -0.2524334788322449, 0.1811009645462036, 1.2475184202194214, -0.8923535943031311, -1.6437228918075562, 1.1283842325210571, 1.846508502960205, 0.3695079982280731, 0.34611597657203674], [0.3621106445789337, -0.06938419491052628, -1.1241377592086792, 0.6198076605796814, -0.38961681723594666, -0.4927181899547577, 0.7493975162506104, 0.23408201336860657, -1.5641765594482422, 1.6528732776641846, 0.15421736240386963, 1.1928105354309082, 0.5278765559196472, 2.3355605602264404, -0.12454916536808014, -0.058980751782655716, 0.45846399664878845, -1.8448423147201538, 1.4847739934921265, 0.1980413794517517, 0.263124942779541, -0.9445675611495972, 1.0813052654266357, -0.7232258915901184, 0.7447372674942017, 0.15167702734470367, 0.33942902088165283, -2.0633902549743652, 0.6946454048156738, -0.3970474600791931, -0.48207882046699524, -0.12488400936126709, -1.2076382637023926, -0.6463228464126587, -1.2011343240737915, 0.5102927088737488, -0.9339747428894043, 0.10696563869714737, -1.0689880847930908, 1.344183087348938, 0.46059346199035645, -0.6496005654335022, -0.3757585287094116, -2.756471633911133, -1.4043304920196533, -0.7108500599861145, 0.8909386396408081, 0.5161116123199463, -0.8863545656204224, -0.518272876739502, 1.1206932067871094, -0.5717130899429321, 1.6810877323150635, 0.7537243962287903, 0.5921782851219177, -0.33770182728767395, 0.7235748767852783, 0.2745453119277954, -0.8266781568527222, 0.576370894908905, 1.076399326324463, -0.40279683470726013, -0.782489538192749, 1.8344320058822632], [-0.22438944876194, -0.16869968175888062, 1.1104499101638794, -0.869167149066925, 0.06219654902815819, 0.4679105877876282, -0.22759303450584412, 0.035739656537771225, -0.8142125606536865, -0.621021568775177, -0.5540475845336914, 0.2545577883720398, 0.7376269102096558, 3.4716429710388184, 1.0056639909744263, 0.22834621369838715, -1.1331666707992554, 1.9727849960327148, 0.9166873097419739, 1.9362131357192993, 1.2844754457473755, -1.165470838546753, 1.0342365503311157, 0.6162504553794861, 1.0043708086013794, -0.9685503244400024, 1.2635527849197388, -0.15084132552146912, 0.6656199097633362, -0.3570716083049774, 1.4495657682418823, -0.3189795911312103, 0.1355619728565216, -0.4177076816558838, 1.7541018724441528, -1.2341370582580566, 1.825742483139038, 0.6589281558990479, -0.5024825930595398, 1.338981032371521, -0.7589388489723206, 1.5580919981002808, 0.2820615768432617, -1.3549904823303223, -0.8405407667160034, -0.7373510003089905, 0.4457366466522217, -0.22450150549411774, -0.7510747909545898, -0.5503519177436829, -0.9839306473731995, -0.06459003686904907, -0.5217848420143127, 0.13642007112503052, -1.0801818370819092, 0.2660045027732849, -0.3802306056022644, 0.3888090252876282, -0.6286234259605408, 1.428016185760498, 0.3918933570384979, 1.2418420314788818, 0.4606216549873352, -1.3334294557571411], [0.31798237562179565, 0.0659203827381134, -0.9279582500457764, 0.507053554058075, 0.7843482494354248, 0.18462948501110077, -2.554110288619995, -0.4667470455169678, -0.691213846206665, 0.5010853409767151, 0.20979973673820496, -1.2487013339996338, 0.029958680272102356, 0.08141926676034927, -0.12818825244903564, 0.06545793265104294, -0.4852053225040436, -0.25498896837234497, 1.136204481124878, 0.03410077095031738, -0.26853659749031067, 0.08596596866846085, 0.2430085390806198, -0.7527589797973633, -0.16770629584789276, 0.3456641733646393, -0.9581198692321777, -0.978850245475769, -1.388777494430542, -0.8629847168922424, -1.757051944732666, 1.5792423486709595, -0.3570227026939392, 0.3268083333969116, -0.3441433906555176, -0.291486531496048, -1.6672160625457764, 1.8437132835388184, 0.48877865076065063, -0.9327227473258972, -0.2602207362651825, 1.095906138420105, -2.3510196208953857, -2.3728394508361816, -0.6951707005500793, -0.09779403358697891, -0.1437031775712967, -0.3200152516365051, 0.7149192690849304, -0.49085381627082825, -0.4084891378879547, 1.698216199874878, -0.3010651469230652, 1.2836304903030396, 1.5965771675109863, 0.15999262034893036, -0.26861682534217834, -0.4990246593952179, 1.5442538261413574, -0.5578703880310059, -0.47955113649368286, -1.4582610130310059, -0.6823030710220337, -1.1102874279022217], [-0.18003568053245544, -0.7202930450439453, 0.3072965145111084, -1.0658906698226929, -0.9149672985076904, -0.21392948925495148, 0.7390870451927185, -0.3303731679916382, -0.6763021945953369, 0.8132874965667725, 0.6990155577659607, -1.0336359739303589, -0.0687059536576271, 0.6560996174812317, -0.4433608651161194, -1.4808449745178223, 1.5094841718673706, 0.4726615250110626, -0.02265622280538082, 0.33600887656211853, 1.3985590934753418, 1.7836503982543945, -0.8968572020530701, 0.578779399394989, -0.626167893409729, 1.3275394439697266, 0.3149878978729248, 0.13036689162254333, 0.262416809797287, -1.834931492805481, 1.3036372661590576, 2.075930118560791, -1.877469778060913, -0.283715158700943, 0.2209279090166092, 0.23771342635154724, -0.2595975995063782, 0.8294289112091064, -0.12046733498573303, -0.8910617232322693, -0.34161123633384705, 0.24305939674377441, 0.5113100409507751, -0.1720454841852188, -0.1755926012992859, -1.4053740501403809, 0.6344484090805054, 1.0427521467208862, 0.19855061173439026, -1.5669384002685547, 0.6543110609054565, -1.2500852346420288, -0.7990010976791382, 1.2247556447982788, 1.820975422859192, 1.0431110858917236, 0.9196025133132935, 0.6811088919639587, 0.32632410526275635, 0.8483301401138306, 0.2662065327167511, 2.0343024730682373, -0.7005279660224915, 0.493900865316391], [-0.2706405520439148, 0.37736713886260986, 0.07563714683055878, 1.0615919828414917, -0.6379143595695496, 0.050266142934560776, 0.6724192500114441, 0.16105812788009644, -0.7080006003379822, 0.17184165120124817, -0.5343366265296936, 0.6721497774124146, 0.5965477824211121, -1.601720929145813, -0.24759796261787415, 1.0417027473449707, 0.41170206665992737, 0.2970460057258606, 0.9319594502449036, 0.8594779968261719, -0.14203980565071106, 0.7237611413002014, 0.056689437478780746, 2.7286972999572754, -0.3409373164176941, -0.20858722925186157, -0.6334540843963623, 1.2096768617630005, -0.8618515133857727, -1.5419740676879883, 0.7064141035079956, 2.1142871379852295, 0.23592296242713928, 0.09658671915531158, -0.6496462821960449, 0.503114640712738, -0.2598898708820343, 1.0630367994308472, -1.3911319971084595, -1.2777737379074097, 0.9223592877388, -0.47747936844825745, -1.4298514127731323, -0.4907395541667938, 0.8804557919502258, -0.02669261023402214, 1.9284626245498657, 0.16346168518066406, -0.08917776495218277, -1.0803834199905396, -1.1666898727416992, -0.19066154956817627, 1.3094943761825562, -0.8882676362991333, -1.2012287378311157, 2.0486414432525635, -0.30607643723487854, -1.6465011835098267, -1.1110033988952637, 1.0639097690582275, 0.5516441464424133, -0.8089625239372253, 0.35567712783813477, 1.1376217603683472], [-2.222621440887451, 0.6968599557876587, -0.29806143045425415, 1.3263188600540161, -1.0198359489440918, -1.7306259870529175, 0.359554648399353, -0.14535285532474518, -0.16784363985061646, 1.3189435005187988, -0.6914283633232117, 0.8875023722648621, 0.6176141500473022, 0.44525662064552307, 0.0260548684746027, -0.5825179219245911, 0.10395949333906174, 0.431061327457428, -1.1091957092285156, -0.9839882254600525, -0.038571860641241074, 0.6883587837219238, 0.6496639847755432, -1.6109607219696045, 0.6548362374305725, 0.3625301420688629, 0.7079740166664124, -0.5575028657913208, 0.9666563272476196, 0.4166070222854614, -1.0148756504058838, 0.9395058155059814, 0.6671335101127625, 0.4290933310985565, 0.32272136211395264, 0.48380300402641296, 0.529585063457489, 0.43193429708480835, -0.143915057182312, -0.1794971525669098, -1.6612824201583862, 0.003774502081796527, 1.9931515455245972, -0.3465763032436371, -2.0057716369628906, 0.11303512752056122, 1.263386607170105, -2.601210355758667, 0.4496280252933502, 0.14030422270298004, 0.7340607047080994, 0.4805912971496582, 0.7089076042175293, -0.576466977596283, 0.618343710899353, -0.14133836328983307, -1.2273341417312622, -0.9636486172676086, 0.32674816250801086, 0.42400649189949036, 0.7570945024490356, 0.8996336460113525, -0.07444839179515839, -0.8329589366912842], [-2.491142511367798, -1.4650871753692627, 0.3530506193637848, 0.5613402724266052, -1.721348524093628, -1.7996925115585327, -1.979477882385254, 0.3022749423980713, 0.5562540292739868, 1.3736493587493896, 0.20503918826580048, 0.00835457630455494, -0.4638523459434509, -0.5986002683639526, -0.666019856929779, -0.478748083114624, -0.00149011448957026, 0.9010846614837646, -1.3126856088638306, -0.8256801962852478, 0.02124667540192604, 1.1307145357131958, -1.2018004655838013, 0.08720073103904724, -1.2786836624145508, -1.5611588954925537, -0.9233002662658691, 0.815568208694458, 0.17248916625976562, -0.11202464252710342, 0.9788939356803894, -0.1859937608242035, 2.413196563720703, 0.8033033609390259, -1.4201382398605347, -2.1390905380249023, 0.9084866046905518, -0.871623694896698, -1.1765440702438354, 0.4420866072177887, -0.40004751086235046, 0.7237908840179443, -1.5105863809585571, -0.2583433985710144, -1.0123951435089111, -0.8689039945602417, -0.4065972566604614, -0.11594178527593613, -1.0869635343551636, -0.5343455076217651, -1.212954044342041, 0.2638813853263855, -0.7748854756355286, 0.03631066158413887, -0.5577424764633179, -0.7610175609588623, 0.18629203736782074, -0.9942973852157593, -0.7811391949653625, -0.6274979114532471, 0.036744456738233566, -0.26894739270210266, -0.09174619615077972, -0.021062880754470825], [-0.4129331707954407, 1.602824330329895, -1.2146037817001343, 0.8846412301063538, 0.8806098103523254, -0.6101654171943665, -0.5260867476463318, -1.1080760955810547, 0.3009497821331024, 1.8649860620498657, -0.9142128229141235, 1.6521838903427124, -0.6082265973091125, -0.8851603865623474, 0.8610663414001465, -0.1179925948381424, -1.2331337928771973, -0.0477275550365448, -1.8691306114196777, 1.93748140335083, 0.31641483306884766, 0.6355822682380676, 0.10373526811599731, 0.22439061105251312, 0.3686678409576416, -0.4979347884654999, 0.6520386338233948, -1.1386791467666626, 1.3685429096221924, -0.16747243702411652, 0.16771820187568665, -0.939864456653595, 0.4257173538208008, -0.776019811630249, -0.4641116261482239, -1.1683650016784668, -0.40518617630004883, -0.37764811515808105, -2.0065758228302, 0.5332214832305908, 1.4694299697875977, 0.04364190250635147, 0.12146789580583572, 1.8646559715270996, -0.6520918011665344, 0.22962649166584015, 0.4722978174686432, -0.468271940946579, 0.24278263747692108, -0.15009549260139465, -0.5002050399780273, -0.3676379323005676, -1.8469280004501343, -0.2337934672832489, 2.9542768001556396, -0.09595362097024918, -0.5850013494491577, -0.0021646542008966208, -0.4992406368255615, -1.9479292631149292, 0.06131899356842041, 0.07206331193447113, 0.6534856557846069, -0.052344854921102524], [-0.567361056804657, 0.19441251456737518, 2.1497998237609863, -0.9348298907279968, 0.6494684815406799, 0.07423694431781769, -1.981326937675476, -0.8372501134872437, -0.15383584797382355, -0.8755334615707397, -1.034018635749817, -0.5695655941963196, -0.8682419061660767, 0.4191490113735199, -1.3128026723861694, 0.19850291311740875, 1.0390206575393677, 1.7633103132247925, 1.665236234664917, 0.3684329390525818, 0.3666176497936249, -1.7164225578308105, -0.5997970700263977, 0.17321571707725525, 0.423198938369751, 0.7129411697387695, 1.5161463022232056, -1.2926595211029053, 0.7808354496955872, 0.8245092630386353, -0.9838170409202576, -0.5089096426963806, 0.8363769054412842, 0.05357604846358299, -0.22640089690685272, 0.7034282684326172, 0.049830880016088486, 1.0122747421264648, -0.5626271367073059, 0.3850454092025757, -0.7330831289291382, 0.8039567470550537, 0.9349582195281982, -1.7572858333587646, -0.4580652415752411, -0.9916250705718994, -0.4131723940372467, 0.001977301901206374, 1.2916429042816162, -0.2747516334056854, 1.0919169187545776, -2.3731184005737305, -1.106114387512207, 1.7855093479156494, 0.1632673591375351, 0.8838699460029602, -0.28402164578437805, -1.1579207181930542, -0.7894216179847717, -0.9130850434303284, -0.4652995467185974, 0.8119468688964844, -0.7628535032272339, -1.517168402671814], [-0.2528708875179291, -2.113391160964966, -2.0621800422668457, -2.6245288848876953, -0.024126999080181122, -1.1754595041275024, -0.48876988887786865, 1.8195244073867798, 0.8462713956832886, 0.6506062150001526, -0.501177191734314, -1.391361951828003, -1.065793514251709, 0.5854707360267639, -0.8269511461257935, -1.6722350120544434, 0.6100916862487793, 0.603508472442627, 1.6627353429794312, 0.8867545127868652, 1.3062574863433838, 0.0679018422961235, -1.977451205253601, 0.15939930081367493, -0.14307209849357605, -0.463395357131958, -0.3715705871582031, 0.9510794281959534, 0.9964668154716492, -0.6774572134017944, -0.920335054397583, -0.35516121983528137, -0.06814442574977875, 2.853600025177002, -0.41196635365486145, 1.251131296157837, -0.9205716252326965, 1.9644060134887695, -0.3784146010875702, 0.9488301873207092, 0.012944061309099197, -0.9009006023406982, 1.6188164949417114, -0.634522557258606, 0.3516036868095398, -0.2751188278198242, -0.2818855047225952, 0.6079463362693787, -0.007893619127571583, 1.4240988492965698, -0.37177717685699463, 0.9139630794525146, 1.090138554573059, 0.9240682125091553, 0.687263011932373, 1.4427642822265625, -0.02594313584268093, -0.6391281485557556, 0.5881005525588989, -0.8943619132041931, -0.15864197909832, 0.7412814497947693, -0.7293584942817688, -0.9450697898864746], [1.7500884532928467, 1.7592486143112183, -0.6446912288665771, 1.2078347206115723, 0.6741427779197693, -0.5476709604263306, 0.025763727724552155, 2.0409960746765137, -0.3368695080280304, -0.94310462474823, 1.5575571060180664, 0.3811783790588379, -0.19158940017223358, -1.1011369228363037, -0.42887839674949646, 1.103419303894043, -1.3764265775680542, 0.3860916793346405, 1.1957008838653564, 1.8348015546798706, -0.061679285019636154, 0.15027117729187012, 1.7814873456954956, -1.6799951791763306, 0.622651219367981, -0.6169490814208984, -1.531760811805725, 0.23410339653491974, -0.6442770957946777, -2.7863247394561768, -1.108317255973816, -1.123127818107605, -1.2100460529327393, 0.34523895382881165, 0.832643985748291, 0.30300769209861755, -0.8860780000686646, -1.717290997505188, 0.09979614615440369, 0.016345227137207985, 0.38008934259414673, -0.22054092586040497, 0.39256301522254944, 0.5418031215667725, -0.9603578448295593, 0.5526875257492065, -1.3280014991760254, 0.20304429531097412, -2.0413386821746826, -1.0671441555023193, -0.7782555222511292, 0.6756620407104492, -0.31559231877326965, 0.7402429580688477, 0.20348310470581055, -0.04902420938014984, 0.44459548592567444, 0.08845669776201248, 0.9536468386650085, -0.8005965352058411, 0.839755654335022, -0.23346452414989471, -2.0936946868896484, -1.0778077840805054], [0.2937889099121094, -1.105808138847351, 0.17207427322864532, 0.37486737966537476, -0.28756022453308105, -0.4917624890804291, 0.5732294917106628, -0.6418139934539795, 0.3915851414203644, 1.7185348272323608, 0.3362762928009033, 1.43019437789917, 0.17955133318901062, -0.11432500183582306, -0.6196939945220947, -0.21011461317539215, 0.24327076971530914, -2.0174560546875, 0.4305715262889862, -1.1370058059692383, -0.8764574527740479, -0.536120593547821, 0.2190844863653183, -0.6578856706619263, 0.0768565833568573, -0.026101363822817802, -0.5942836999893188, 1.5778926610946655, -0.5669134259223938, -2.324385404586792, -2.2550835609436035, -1.660048246383667, 1.894419550895691, -1.490781545639038, -1.209480881690979, 0.7191405892372131, -0.5524048209190369, 0.15687638521194458, 0.24249553680419922, 0.4727490544319153, -0.04841010272502899, -0.7385356426239014, 0.3260432779788971, 0.7154673933982849, 1.193461298942566, -0.6174799799919128, 1.9853670597076416, -1.384554147720337, -0.7414166927337646, -0.47588783502578735, -0.9686909914016724, 0.3145153522491455, -2.500626564025879, 0.03689241409301758, 0.32329514622688293, 1.2521761655807495, 0.15215502679347992, -1.3210567235946655, 1.0671579837799072, 0.6851764917373657, 0.4362407922744751, 0.16904161870479584, -0.36996200680732727, -0.4446425139904022], [-0.41284170746803284, -0.5696025490760803, -1.263498067855835, -0.9045419692993164, 2.119365692138672, -0.9930221438407898, 0.6928479671478271, 0.41955575346946716, 0.14838755130767822, 1.307634711265564, 0.3140791952610016, -0.6329288482666016, -1.4585473537445068, -0.06632667034864426, 1.0927560329437256, 0.9953721761703491, -0.3689122200012207, 0.0026035388000309467, -1.4030746221542358, -1.0841114521026611, -0.24556642770767212, -0.2709049880504608, -1.158284068107605, -0.6752271056175232, -0.8185185194015503, -1.109775185585022, -1.8298660516738892, -0.7718451023101807, 0.7082500457763672, -0.7616713047027588, -1.058420181274414, -0.3990769684314728, -0.633354127407074, 1.3448634147644043, 1.5986093282699585, 1.1714589595794678, -0.6257601976394653, 2.55035400390625, -1.0228008031845093, -0.5747440457344055, 0.7300028204917908, 0.9322211742401123, 0.332792192697525, -1.3700525760650635, 1.6316215991973877, -0.06164855882525444, -0.06519097834825516, -1.074544072151184, 0.7936941385269165, -0.2451920211315155, 1.4860786199569702, -0.17195038497447968, 1.706918478012085, 0.3640736937522888, 0.2620640695095062, 0.9974625706672668, 0.12342154234647751, -0.7659723162651062, 0.19339996576309204, 0.22695648670196533, 0.16572700440883636, -0.07692935317754745, 0.5380570292472839, 0.0351153165102005], [0.6440519690513611, 0.6823661923408508, -0.030106283724308014, 0.36511456966400146, -0.22528454661369324, -2.316505193710327, -0.8359275460243225, -0.9636598229408264, -2.024498462677002, -1.0119374990463257, -0.5440934896469116, -0.6705576777458191, -1.960254430770874, 0.9923592209815979, 1.8187154531478882, -1.1227234601974487, 0.7524111270904541, -0.47420164942741394, 0.2864513099193573, -0.32614415884017944, -0.3153446912765503, 0.6930075287818909, 0.21013283729553223, -2.2857062816619873, 1.1735631227493286, 0.7884500622749329, 0.4867730140686035, -1.207452416419983, 0.7091339230537415, 1.3145629167556763, 0.03518044576048851, 3.0280773639678955, 1.3676213026046753, -1.0477055311203003, -1.2831213474273682, 0.9524837136268616, -0.7604231834411621, 0.3776852786540985, -0.733862042427063, -0.9617137908935547, 1.0423392057418823, -0.2886364161968231, -0.18507392704486847, -1.212517499923706, 1.1784539222717285, 0.3832186460494995, -0.09362929314374924, -0.24997001886367798, 1.309133529663086, 0.26247209310531616, 0.8788118362426758, -0.3348407447338104, -1.8657366037368774, 1.1518911123275757, -0.13305819034576416, -1.346522569656372, -0.10740502178668976, 0.1229131892323494, -0.8001745939254761, 0.5590147972106934, -0.5963361859321594, 2.1260287761688232, 0.18117576837539673, -3.585341691970825], [-2.203516721725464, -0.776348888874054, 0.7787964940071106, -1.333950161933899, -0.6405661106109619, -1.1843432188034058, -0.6103495359420776, 0.03979984670877457, 0.38980409502983093, -1.2187169790267944, 0.39025112986564636, 0.4503359794616699, -0.2740209996700287, -0.060671474784612656, -0.5137892961502075, -0.7689374685287476, -0.4421132802963257, -2.561314582824707, 0.0278745349496603, -0.3624184727668762, 0.7360967993736267, -0.5146434307098389, 1.9444451332092285, -0.18928304314613342, 1.20314359664917, -0.9359855651855469, 0.948504626750946, 0.6908193230628967, -1.1522984504699707, 1.1304545402526855, -1.0197299718856812, -2.7681446075439453, -0.7383292317390442, 0.4463750720024109, -1.2851020097732544, 1.8924247026443481, 0.8978054523468018, -0.023364560678601265, 0.2080228626728058, 0.9270498752593994, -0.030088389292359352, 0.5591328144073486, -0.37638238072395325, 0.8111831545829773, -0.25410035252571106, -0.11139221489429474, 0.07719849795103073, 0.3452129662036896, 0.7355721592903137, 0.7073506712913513, 0.7107846140861511, 0.7500745058059692, -0.7103550434112549, 0.5500470995903015, 1.2299139499664307, -0.5898805260658264, 0.6543745398521423, -0.443739116191864, 0.7928482890129089, 0.452474981546402, 1.8572098016738892, -1.0049697160720825, -0.09874589741230011, 1.4153203964233398], [0.3655761480331421, -0.6717879772186279, -1.778012752532959, -0.9745078682899475, 0.9686456918716431, 1.7439064979553223, -1.358697772026062, 1.1227892637252808, -0.40049824118614197, 0.310079962015152, -0.043273188173770905, -1.2116193771362305, -2.1016159057617188, 2.0483789443969727, -0.8351594805717468, -0.7609977722167969, -1.0891544818878174, 1.0329022407531738, -0.1380084902048111, -1.6290050745010376, 0.20761698484420776, 1.9600118398666382, 1.2902143001556396, 0.741778552532196, 2.370546817779541, -1.145711898803711, 1.1328308582305908, -1.4269225597381592, -1.1978709697723389, 0.1404905915260315, 0.26245930790901184, 0.9518746733665466, 0.3399997055530548, -1.165307879447937, 0.13368453085422516, -1.059943675994873, 0.7905813455581665, 0.5698738098144531, 0.09861306846141815, -0.5907372236251831, 1.431994080543518, 0.26541978120803833, -0.4124770760536194, -0.15840765833854675, 0.15377138555049896, 0.402936726808548, -1.656083583831787, 0.5384661555290222, -0.09800844639539719, -0.3798191249370575, 1.3362860679626465, 0.696076512336731, 0.5227991938591003, 0.24026906490325928, -0.7767655849456787, -1.1942596435546875, -0.3599165678024292, -0.10990747809410095, 0.8015374541282654, -1.4446791410446167, 1.6871449947357178, 0.21402668952941895, 0.14009994268417358, 0.035372644662857056], [0.44435060024261475, 0.09640063345432281, 0.6849846839904785, 0.552154541015625, 0.22076503932476044, -0.265203058719635, 1.5237687826156616, 0.7550861239433289, 0.9406978487968445, 0.6554536819458008, -0.6247079372406006, -1.0197649002075195, -0.5947641134262085, 0.37904924154281616, -0.6930409073829651, -0.635478675365448, -0.9971616268157959, 0.46018078923225403, 0.37814152240753174, 1.0920500755310059, 0.1036033034324646, 0.22704321146011353, -0.9206393957138062, -0.3382169008255005, -1.06333327293396, 0.7641572952270508, -1.1045736074447632, 1.2117480039596558, 1.0062743425369263, -1.8291553258895874, -0.28634658455848694, 0.7185558676719666, 0.8299240469932556, 0.38808473944664, -0.2457682341337204, 0.2524346709251404, -1.8462337255477905, -0.3807714879512787, -0.7113410234451294, -0.30801862478256226, 1.953595519065857, -1.2059320211410522, -1.1035386323928833, -0.016801657155156136, 0.6825132369995117, 0.30497947335243225, 0.21925610303878784, 0.8974167704582214, 1.3793718814849854, -0.37985101342201233, 1.0165437459945679, 0.44952431321144104, 0.8535789847373962, 1.0029536485671997, 1.7963299751281738, -1.7721816301345825, -0.8115519881248474, 0.5504107475280762, 0.022152021527290344, -1.5327459573745728, -0.45782139897346497, 0.29275715351104736, -1.5919113159179688, 2.270369529724121], [1.0083606243133545, 1.0076062679290771, -0.40848878026008606, 1.3684979677200317, -1.0220073461532593, -1.6227084398269653, 0.2163785845041275, -1.7118109464645386, -0.3602074086666107, -0.2790890634059906, -1.4895013570785522, -0.5874799489974976, 0.48405230045318604, 1.9843019247055054, -0.1428425908088684, 1.33634614944458, -1.610176682472229, -0.6704469323158264, 0.30294275283813477, -0.9966601133346558, 1.141855239868164, -1.0149295330047607, -1.1292937994003296, -0.7386791706085205, -0.45786452293395996, -0.3779611587524414, -0.08335606008768082, 1.453583836555481, -1.36408269405365, 0.6719860434532166, -0.6042805910110474, -0.7846507430076599, 1.1414289474487305, 0.41407307982444763, -0.7710109949111938, -2.2002370357513428, -0.4181041717529297, 0.3437175750732422, -0.9381936192512512, -0.02629811502993107, 0.6614728569984436, 0.7740176916122437, -0.9777917861938477, 0.3966943025588989, -1.0059410333633423, 0.6902499794960022, 2.1009318828582764, 1.435351014137268, -0.29652202129364014, 1.4289966821670532, -0.7141695618629456, 0.19390547275543213, -1.0803449153900146, -0.8873417973518372, -0.5307565927505493, 0.6346037983894348, -0.9692539572715759, 0.6391270756721497, -1.6152812242507935, -0.372030645608902, 0.542384147644043, -0.09050676226615906, 0.38581982254981995, -1.2545583248138428], [-0.052218902856111526, 1.2935363054275513, -0.7175782322883606, -0.18407104909420013, 1.139990210533142, -0.5552913546562195, 0.2531212568283081, 1.661006212234497, 0.8847576975822449, 0.010738896206021309, 0.8352985978126526, -0.5052105188369751, 1.2030223608016968, -1.1469844579696655, -0.7985746264457703, 1.4994550943374634, -1.3419113159179688, 0.1489885300397873, 0.22867050766944885, 0.8249324560165405, 1.0124552249908447, 0.044905513525009155, 0.18788494169712067, -0.629148542881012, 1.2752583026885986, -1.118140697479248, -0.6089221835136414, 0.026902005076408386, -1.152064323425293, -0.351507306098938, 0.6274399161338806, 0.060353461652994156, -0.5133511424064636, -1.03739333152771, 0.5471743941307068, 0.9513143301010132, 1.1893812417984009, 3.0790653228759766, -1.4806965589523315, -0.5275941491127014, 0.9416690468788147, 0.13096411526203156, 0.24872712790966034, 1.7461891174316406, -0.3461877405643463, -0.44747811555862427, -0.7376258969306946, -0.3723750710487366, -0.1362554281949997, -0.5451165437698364, -1.0065683126449585, 1.8522108793258667, -0.4780627489089966, 0.43332189321517944, -0.2835370898246765, 1.5605897903442383, -0.24512264132499695, -0.18603205680847168, -1.6065715551376343, -1.6887595653533936, -1.6613540649414062, 0.6672886610031128, -0.16128681600093842, -0.9665630459785461], [-1.7830976247787476, 1.6594675779342651, -0.8453497290611267, -0.6873436570167542, 0.7281920313835144, 0.06704400479793549, -0.957971453666687, -2.3422131538391113, -0.034892309457063675, 0.6652113795280457, 0.040391456335783005, 0.38371363282203674, 0.08539982885122299, -0.7172030806541443, 0.9783485531806946, -0.2585030496120453, -1.4255249500274658, 0.3849044144153595, -0.8390987515449524, -0.4980674684047699, 0.7457475066184998, -1.5607564449310303, -0.960527241230011, -0.8651021718978882, 0.6659074425697327, 0.436628133058548, 1.2978547811508179, -0.47927528619766235, 2.0331907272338867, -1.7954295873641968, -1.65230131149292, 0.3936688005924225, 0.20652590692043304, 1.319627046585083, 0.21305377781391144, -1.700925588607788, -0.43278974294662476, -2.026716947555542, -1.8755930662155151, 0.4724254906177521, 0.0864756777882576, -0.7184786200523376, -0.1214454397559166, -0.15060406923294067, 1.313145637512207, -1.754839301109314, 0.9484094977378845, 0.368603378534317, -0.2542097866535187, -0.9328454732894897, -0.18565107882022858, 1.5312460660934448, 1.2488653659820557, -0.34563323855400085, -1.3225315809249878, -0.8294268250465393, -0.6564052104949951, -0.42464467883110046, -1.347375512123108, 1.182698130607605, 0.6597194671630859, 0.11809061467647552, -0.6456441879272461, -1.6573907136917114], [0.042925670742988586, 0.5624338984489441, -0.06818775832653046, -0.9450625777244568, 0.1317788064479828, 0.6810430288314819, 1.9286856651306152, 2.030576705932617, 0.6461362838745117, -0.13084375858306885, -0.5281012654304504, -0.06601044535636902, 0.7553715109825134, -0.7519407272338867, 0.35837212204933167, -1.0264004468917847, -0.4881170094013214, -0.5313459634780884, 0.3767600357532501, 0.7399272322654724, -0.36590513586997986, -1.4076433181762695, 0.8929929137229919, 0.19106268882751465, 1.0312695503234863, 0.7646411061286926, -0.18472154438495636, -0.5286437273025513, 0.17728696763515472, -0.3814089298248291, 0.3942555785179138, -0.270474374294281, -1.4131730794906616, 0.059578895568847656, 1.4487297534942627, 0.9321407079696655, 0.5014477968215942, -0.642409086227417, -2.486680269241333, -0.5215353965759277, -0.12914042174816132, 0.4715898334980011, -0.2636808156967163, -1.796278476715088, -0.7996342182159424, -1.4462003707885742, -0.7313455939292908, 1.4151654243469238, 1.5304574966430664, -0.8306252956390381, -1.6038527488708496, -0.09288335591554642, 0.02323726750910282, 0.5306000113487244, -0.516427218914032, -0.21898850798606873, 0.5797726511955261, -0.48303332924842834, 0.978554368019104, -1.4775989055633545, -1.0267025232315063, -0.3521639108657837, -0.22075527906417847, -1.55482017993927], [0.82147216796875, -0.0029428862035274506, 0.09171074628829956, 1.406817078590393, 1.9176099300384521, 1.0244485139846802, -0.988725483417511, 1.7891637086868286, 0.5564339756965637, -0.6059570908546448, 1.5698187351226807, 0.18089358508586884, -0.044509608298540115, -1.9262737035751343, -0.06489665806293488, 2.2664175033569336, 1.3113150596618652, -0.05878904089331627, -0.14912524819374084, 0.5761451125144958, 0.19973669946193695, 0.5998287200927734, -0.4746803939342499, 0.32117006182670593, -0.10287970304489136, 0.8932141065597534, 0.6731496453285217, -0.07050511240959167, 0.3097221851348877, 1.186165452003479, -1.2398608922958374, -0.384869247674942, -0.6758100986480713, -0.5683821439743042, -1.5028644800186157, 1.0222842693328857, 0.7867078185081482, 0.6795156598091125, -0.6040934920310974, -0.07404287159442902, 0.6031363606452942, 0.029654553160071373, 0.20159763097763062, 0.07087001204490662, 0.5322755575180054, 1.4290341138839722, -0.3811689615249634, -0.719393253326416, 2.423614501953125, -2.6603124141693115, -0.5668528079986572, 1.0096532106399536, 1.3085503578186035, -0.25394755601882935, 0.6531201601028442, -0.30298152565956116, -0.26841771602630615, -0.04166799783706665, -0.38827943801879883, -1.1489187479019165, -0.9656543731689453, 0.4148065745830536, -1.4400122165679932, -0.3300229012966156], [0.10800302028656006, -0.9257987141609192, -0.710366427898407, 2.31974458694458, 0.915747880935669, -0.22672753036022186, 0.539830207824707, -1.2789422273635864, 0.26747140288352966, -1.410279393196106, -0.2256740927696228, 0.03365374729037285, 0.2510557174682617, -1.198035478591919, 0.20921741425991058, -0.23873043060302734, -0.934288501739502, -0.6596114635467529, 0.444398432970047, -0.5022914409637451, -2.0050127506256104, 0.7982600927352905, 0.22733406722545624, 0.7894109487533569, 0.5442826151847839, 1.268715262413025, -1.95246160030365, -0.5186723470687866, -0.8816131353378296, -1.7794277667999268, -0.3863908648490906, 0.10746665298938751, -0.24975024163722992, -0.2183190882205963, -1.1040419340133667, -0.17444050312042236, 0.8321815729141235, 0.33014607429504395, 0.06405290216207504, 0.7976610064506531, -0.3473769724369049, 0.052268486469984055, -0.03163773939013481, 0.46138858795166016, 1.6913832426071167, -0.8623219728469849, -0.5384776592254639, -0.224532350897789, -1.521368384361267, -0.822893500328064, 0.8658590912818909, -0.003264942206442356, 0.5920940041542053, 1.5292534828186035, 0.40425360202789307, -0.06817296892404556, -0.7725732922554016, -0.13877850770950317, 0.31901538372039795, 0.23386304080486298, -1.137406349182129, 0.4717361032962799, 2.1361401081085205, 1.2928224802017212], [0.8123346567153931, 0.3301679491996765, 0.974094033241272, 0.6086004376411438, 0.8319888114929199, -0.7401926517486572, -0.20192690193653107, 0.3807576596736908, 0.4012710452079773, 0.19570642709732056, -1.7400604486465454, 1.3699836730957031, 0.6330839395523071, -0.3299422860145569, -0.2573547661304474, 1.1747850179672241, 1.2481000423431396, -1.2130504846572876, 0.4453393518924713, 1.0639573335647583, -0.19081580638885498, -0.9504335522651672, 0.21579097211360931, 1.402241826057434, -0.30923908948898315, -2.0994112491607666, -0.5193235278129578, 0.32422149181365967, 0.6961080431938171, 0.10755319148302078, 0.36444321274757385, 0.3138631284236908, -0.17856359481811523, 0.18984843790531158, 0.4481165111064911, -0.4678622782230377, -0.03305866941809654, 0.13630253076553345, -0.7593666911125183, -0.5500989556312561, -0.4422760605812073, -0.5400905013084412, 0.12362220883369446, 0.0078512504696846, -0.3135235905647278, 0.34099435806274414, -0.1745220124721527, -0.559889018535614, -0.3966695964336395, 0.9635204076766968, 0.896856427192688, -0.40838032960891724, -0.5287071466445923, -0.3088233172893524, -1.6558808088302612, 1.0633336305618286, -1.61008882522583, -1.2438304424285889, -0.8736933469772339, 1.366220235824585, 0.01064272690564394, -0.15605324506759644, 0.10862001776695251, -0.9244945049285889], [-0.37374576926231384, -0.10001713782548904, -0.17538940906524658, 0.6664581894874573, -0.2642383277416229, -1.2646582126617432, -0.1929783970117569, 0.7875168919563293, -0.35945865511894226, 0.7717350125312805, -0.15404768288135529, 0.5628042817115784, 0.3928525149822235, 1.3975898027420044, 0.5603559613227844, -0.5215277671813965, 1.395291805267334, 1.5965272188186646, 0.3283200263977051, 0.19374847412109375, -0.7559216022491455, -0.360532283782959, 0.46632230281829834, 1.1605826616287231, 0.16587337851524353, -0.14865709841251373, -0.4840942919254303, 1.5333733558654785, -0.8506045937538147, 2.534025192260742, -0.5830357074737549, 1.9210474491119385, -0.8394251465797424, -2.772226333618164, -1.7457256317138672, 0.16457964479923248, 0.861438512802124, -0.12451039254665375, 0.4215700328350067, -0.47727736830711365, -1.2436271905899048, 1.1059951782226562, -0.46080872416496277, 1.1671924591064453, 0.7971113920211792, 1.4778341054916382, 1.2387264966964722, -0.34627795219421387, 1.0747145414352417, -0.761556088924408, 0.5291154980659485, 0.0038581774570047855, -0.5561661124229431, -0.8175936937332153, 0.7375993132591248, -0.9568904042243958, 0.2486708164215088, 0.3338634967803955, 1.3427945375442505, -0.7603593468666077, 1.0928646326065063, 0.19511619210243225, -0.681736171245575, -0.7950332760810852], [-0.1562800109386444, -1.6243720054626465, 0.1867954134941101, -0.2954821288585663, -0.4646247327327728, -0.26326876878738403, 0.551388680934906, -0.42740631103515625, 1.1682909727096558, -0.5186508297920227, -1.031624436378479, 0.19905999302864075, 0.050829168409109116, -0.26290637254714966, -1.013928771018982, -0.5279139876365662, -0.44210970401763916, -0.9875094294548035, -0.14205466210842133, 1.5124690532684326, 0.14392027258872986, 1.1630359888076782, 2.9225587844848633, 0.671241283416748, -1.1047701835632324, 0.305050253868103, 0.8351601362228394, -0.8158160448074341, 0.6079329252243042, 0.442795991897583, -2.2525131702423096, 0.38112500309944153, -0.12433822453022003, -0.4459097683429718, -1.1775403022766113, 0.04416315630078316, -0.5082036852836609, -1.2445440292358398, 1.4618279933929443, -1.66897714138031, -0.9659867286682129, 0.12177104502916336, 1.257843017578125, -1.117190957069397, -1.0265783071517944, 0.47373056411743164, 0.06606078147888184, -0.936406672000885, 1.5202412605285645, 0.9900138974189758, 0.11708347499370575, 0.4767742156982422, 0.22184668481349945, -1.0913468599319458, -0.020275501534342766, 0.6427049040794373, -1.0663468837738037, -0.8223721385002136, -0.8000332713127136, 0.8694086670875549, 0.4288625121116638, 0.6187101006507874, 0.5938355326652527, 1.0461915731430054], [1.0576496124267578, -0.6564475297927856, -0.6551861763000488, -1.0891140699386597, -1.99612557888031, 0.9013528823852539, 0.7857756614685059, 0.9076676368713379, -1.5673505067825317, -0.4278603792190552, -0.15338262915611267, -0.1735670566558838, -0.7994611859321594, 0.7820339798927307, 1.9819141626358032, -0.9209916591644287, 1.4163981676101685, -0.9347786903381348, -1.4432904720306396, 0.6850992441177368, 1.3838706016540527, 0.4016973674297333, 0.018272241577506065, 1.8448247909545898, 0.3249635100364685, -0.7271350026130676, -1.2484854459762573, 1.270844578742981, 1.8180677890777588, -1.400907039642334, -1.0181870460510254, -0.803223192691803, 0.17799603939056396, 0.6883074045181274, 0.395804762840271, 0.703324019908905, 0.7995195388793945, -1.6424096822738647, -0.9388810992240906, -0.31767919659614563, 0.12827935814857483, -0.6929112076759338, 0.5862941741943359, 0.6305115222930908, 1.9610861539840698, 1.4672274589538574, 0.6797713041305542, 2.123976707458496, 0.7868585586547852, -1.6029678583145142, 0.10594255477190018, 0.004236987791955471, 0.5022522807121277, 0.5622494220733643, 0.04176546633243561, -0.46630892157554626, 0.4367371201515198, 0.4502650499343872, -0.7164217233657837, 0.7758682370185852, -1.321036696434021, -0.20868206024169922, 2.1187946796417236, -0.1761443018913269], [-0.34945327043533325, -0.1788855791091919, 0.7956956624984741, -1.2690348625183105, -2.079098701477051, -0.7393621206283569, -0.5002882480621338, 1.2949678897857666, 1.1444306373596191, -0.22350147366523743, -0.606259286403656, -1.523896336555481, -0.836581826210022, 1.8752923011779785, -0.012546028941869736, -1.2744888067245483, -0.48985981941223145, -0.29008954763412476, -0.9183460474014282, -0.409933865070343, -0.8465099334716797, -0.23113150894641876, -1.7536579370498657, -0.786600649356842, -1.6782172918319702, -0.9327352046966553, -0.8977661728858948, -0.3326736092567444, -1.4102524518966675, 0.04269294813275337, -1.494794249534607, 0.9437139630317688, 0.6362816095352173, 1.098043441772461, -0.07806212455034256, -0.2955206036567688, -0.29168522357940674, 1.3424960374832153, 1.3849436044692993, -0.7393301129341125, 1.0927300453186035, 0.8986614942550659, -0.44997742772102356, -0.279648095369339, -0.584913969039917, 1.232592225074768, -0.5142456889152527, 0.31502458453178406, -0.44049543142318726, -2.1968674659729004, -0.688442051410675, 1.6724462509155273, 0.27544906735420227, -1.5115606784820557, -1.6008797883987427, 1.0768405199050903, -0.9278757572174072, 0.5408312082290649, 1.210645318031311, -0.22189050912857056, 0.27818071842193604, -0.45641353726387024, 0.8858627676963806, -1.5900944471359253], [0.2585109770298004, -0.17297512292861938, -0.23494239151477814, -1.4981215000152588, 0.12286213785409927, 0.23672255873680115, 0.6936576962471008, -2.713897466659546, 0.18678151071071625, -1.1407305002212524, -2.0842413902282715, 0.8580674529075623, -0.008388451300561428, 1.3111859560012817, 0.5184521079063416, 0.7560694217681885, -1.0909178256988525, 0.5680695176124573, 1.0675342082977295, 1.284469485282898, -0.081423819065094, 0.6247854828834534, 1.5298335552215576, -1.0575658082962036, -0.1801488697528839, 0.2547459006309509, -0.08574064821004868, 0.8150405883789062, -0.2758452594280243, -1.6714379787445068, -1.4991724491119385, -0.6545300483703613, -0.7661559581756592, -0.4807834327220917, -0.2266092598438263, -1.7943183183670044, 0.6090201735496521, 0.7182361483573914, 0.7181875109672546, -1.556298017501831, 0.9266019463539124, -0.00909417774528265, 0.4887559115886688, -0.11624036729335785, -0.8910959362983704, -0.24150769412517548, 0.5125914812088013, -1.4025737047195435, -1.1341853141784668, 0.06592173129320145, -2.5601353645324707, 0.09578761458396912, -0.5410565137863159, -0.5351817011833191, -0.042954958975315094, 0.45909038186073303, 0.3140161335468292, -0.3803164064884186, 0.23473656177520752, -0.7546621561050415, 0.3713897168636322, 0.7959877848625183, 1.0784528255462646, 0.2575542628765106], [0.7602730989456177, -0.26559555530548096, 2.6013855934143066, 0.18727032840251923, 0.9398632049560547, 0.10697802901268005, 0.06725812703371048, -0.10804717242717743, 2.0173981189727783, 1.5396184921264648, 0.44957172870635986, -0.4579521715641022, 0.78829026222229, -0.49037134647369385, -0.27986079454421997, 0.6931301355361938, 0.07002998888492584, 0.3198443055152893, -1.3422564268112183, 0.07887661457061768, 1.3835686445236206, 1.1095333099365234, 0.35654333233833313, -1.1130363941192627, 0.24358806014060974, 0.9642373323440552, -0.038077227771282196, 0.8578313589096069, -0.7643077373504639, 0.2764316201210022, -0.3456234931945801, -2.251943349838257, -0.6522750854492188, 0.058503855019807816, -0.7764713764190674, 0.6799800992012024, -1.8792697191238403, -0.45576542615890503, -0.15474626421928406, -1.6593823432922363, -1.4883750677108765, 1.033632755279541, 1.5710015296936035, 1.1216129064559937, 0.5260831713676453, 0.08722425997257233, 0.5088902115821838, 1.806396245956421, -0.36433714628219604, 0.5477228164672852, 0.7367720007896423, 0.018640808761119843, -1.0617992877960205, 1.50855553150177, 1.0599243640899658, 0.8592591881752014, -0.404209703207016, 0.6695975661277771, -0.7374197840690613, 0.10448917001485825, 1.3120191097259521, -0.028539475053548813, -1.3632947206497192, -0.9449527859687805], [1.7930974960327148, -0.03110041841864586, 1.5888855457305908, -1.201013445854187, 0.4109313488006592, -0.7264909744262695, -0.6196985244750977, 0.722903311252594, -0.33561623096466064, 0.8082240223884583, 0.6561373472213745, 3.2875444889068604, 1.0818562507629395, -0.7477201223373413, -1.048943042755127, 0.4024783670902252, 1.2207823991775513, -1.2130320072174072, 0.41278454661369324, 1.622085690498352, 0.2904157042503357, -1.098131537437439, -1.6451319456100464, 0.17395764589309692, -1.9857186079025269, 0.9404135942459106, -1.3557868003845215, -1.1206934452056885, 0.8333134651184082, 0.2202303558588028, -0.16194453835487366, -0.8607038259506226, 0.1810169368982315, 0.060137517750263214, -0.06842364370822906, 0.2473013997077942, -0.6775751709938049, -1.5132813453674316, -0.6271727085113525, 0.26281020045280457, -0.4185113310813904, 0.9094371795654297, 0.5756903886795044, 0.1651802659034729, 1.516900897026062, -0.5207260251045227, -0.1227833479642868, -0.1230112686753273, 0.6528096795082092, -1.1726484298706055, 0.5893898606300354, 0.5221089124679565, -0.26510244607925415, 0.6969608068466187, -2.00862979888916, 1.8095859289169312, -1.1156142950057983, -0.6644659042358398, -0.5728819370269775, -0.30607354640960693, -0.2669861912727356, -0.03508896008133888, 0.930902898311615, 0.14419330656528473], [0.8414367437362671, 0.0833342969417572, -0.7157408595085144, 0.5748361349105835, -0.5507664680480957, 0.5956165790557861, 0.01283202413469553, -0.48369544744491577, 0.5864254832267761, -0.2034536898136139, 3.0359156131744385, 0.8302598595619202, 0.9419390559196472, 0.9531829357147217, -0.7619106769561768, -0.23373661935329437, -0.3198985755443573, -0.13803218305110931, 1.2685734033584595, 1.5086158514022827, -0.3311749994754791, -1.1581134796142578, 0.24900908768177032, 0.9286584258079529, -1.3139071464538574, 0.3291718661785126, -0.6626569628715515, 0.8091393709182739, 0.8646912574768066, 0.1189948096871376, -0.9167652130126953, 0.17981736361980438, -1.1200581789016724, 1.009872317314148, -0.6179893016815186, -1.4476985931396484, 1.4884653091430664, -0.5251904726028442, -0.23536746203899384, 2.6321966648101807, -0.893838107585907, -0.07672538608312607, -1.134874701499939, -0.14641600847244263, -1.283149242401123, -0.8868825435638428, -0.3532790243625641, -1.585029125213623, 0.49024006724357605, 0.35862335562705994, 1.4341492652893066, -0.032732658088207245, -0.474718302488327, 0.5906424522399902, -1.7057863473892212, -0.07084918022155762, 1.8634048700332642, 0.19939260184764862, -0.24777573347091675, 1.0647282600402832, -0.3301376700401306, -0.5092440247535706, 1.8790903091430664, 0.9831876158714294], [0.353181391954422, -1.454300045967102, 1.002213954925537, 0.417614221572876, 0.6468594670295715, -0.24560239911079407, -0.4677636921405792, -0.4658927917480469, 2.1912426948547363, -0.8642397522926331, -0.9461167454719543, -0.17349055409431458, -0.5489276051521301, 0.46393176913261414, 0.6838067173957825, -1.0717676877975464, -0.39691904187202454, -0.444785475730896, 0.5009518265724182, -0.7819944024085999, -0.4504551589488983, 0.5642189979553223, -2.054141044616699, 0.052543334662914276, -0.6745789051055908, 0.4728471040725708, -1.975989818572998, 0.13712507486343384, 1.4147523641586304, 0.38228896260261536, 0.08737580478191376, -0.3753998577594757, 0.6741796135902405, 0.38434553146362305, -0.5948625802993774, 0.7075521349906921, 0.907513439655304, 0.5950478911399841, 0.23344255983829498, -0.6697884202003479, 1.074419379234314, 0.36752814054489136, 0.8793246150016785, -0.37500283122062683, -1.1868442296981812, 0.6658586263656616, -0.48929792642593384, 0.016574203968048096, -0.3786567747592926, -0.19222183525562286, -0.6340288519859314, 2.0742740631103516, -1.9651647806167603, 0.6033180952072144, -0.8793317675590515, 0.8041148781776428, 1.1451740264892578, -0.18862462043762207, -0.27434974908828735, -0.49385902285575867, 0.18105024099349976, 1.0224401950836182, -0.1565716564655304, 0.21115247905254364], [0.4082818329334259, -0.09346096217632294, 0.11857062578201294, -0.14195170998573303, -0.18578587472438812, 1.7735751867294312, -0.13996843993663788, -1.2870970964431763, -0.7183057069778442, -0.013796242885291576, -0.5104060769081116, -0.867832601070404, 0.48363566398620605, 0.6471210718154907, 0.1337243765592575, 0.8062341213226318, 0.8276098370552063, -1.8586726188659668, -0.42751839756965637, 0.21898780763149261, 0.2283838540315628, -1.2006210088729858, 0.7678396105766296, -0.3771801292896271, -0.9017436504364014, -0.8360122442245483, -0.03745366260409355, -0.4149131774902344, 0.8697713017463684, 0.34869012236595154, 1.2117656469345093, 0.30661678314208984, -0.9034054279327393, 0.5725255012512207, -1.7270159721374512, 2.152052640914917, -1.460304617881775, -0.02474784106016159, 0.25505515933036804, -0.9749491214752197, 2.785663604736328, -0.6856868267059326, -0.4698035418987274, -0.6688863635063171, -1.045320749282837, 0.6969238519668579, -0.24759618937969208, -0.1413859874010086, 1.3538544178009033, 1.564365267753601, -1.0233453512191772, 3.2793846130371094, 0.2359858602285385, -0.10941307246685028, -0.39669468998908997, 0.3686347007751465, 1.299289345741272, 0.6320350766181946, -0.7349624633789062, -1.7297403812408447, -1.6616352796554565, 0.008120202459394932, 0.7678800821304321, -0.16859929263591766], [0.22347404062747955, 1.647013545036316, 0.424130380153656, -0.42123615741729736, 1.1480244398117065, 0.5712348222732544, -0.2713867723941803, 1.7073816061019897, 2.558990478515625, -0.11430656909942627, -1.0164541006088257, -0.425561785697937, 0.7782385945320129, 0.19753623008728027, 1.4195854663848877, -0.2324296236038208, -1.131966233253479, 0.12399562448263168, -1.8800026178359985, -0.5455622673034668, -0.4591984748840332, 0.4775506258010864, 1.0681767463684082, 0.9695098996162415, -1.3416521549224854, -0.2291523814201355, 2.1370699405670166, -1.041120171546936, 0.9821049571037292, 1.4580613374710083, -0.5820346474647522, 0.9857110381126404, 0.1035642921924591, -0.001911794999614358, -0.9608392715454102, -0.19921891391277313, -1.5351920127868652, 1.4754278659820557, -1.1066532135009766, 0.7755236625671387, -2.172477960586548, -0.31557872891426086, -0.5454376935958862, 0.46267303824424744, -0.23539738357067108, -0.9992309808731079, 1.6760241985321045, 0.6348652839660645, -0.3188105821609497, 0.9839727282524109, 0.6671661734580994, 0.550553023815155, -1.6809165477752686, 0.46368980407714844, -1.011013150215149, -1.3576830625534058, 0.29947564005851746, -1.1661709547042847, 0.8490601181983948, -0.12744927406311035, -2.4283881187438965, -0.5049504041671753, 0.5463310480117798, 0.18150047957897186], [-0.9180737137794495, 0.9682484865188599, -0.5830325484275818, -0.5121381282806396, 0.09472797065973282, -0.3993232250213623, 0.18745741248130798, 1.5531537532806396, 0.7679218053817749, 0.38797861337661743, -0.6337501406669617, -0.21649640798568726, -0.6860378980636597, -0.6057388782501221, 0.06404223293066025, 0.09278132021427155, -0.9496776461601257, 1.2600082159042358, -0.9136282205581665, -0.19632942974567413, -1.7337093353271484, 0.04086514934897423, 1.3884190320968628, 1.8279181718826294, -0.3326323330402374, 1.5575953722000122, -0.491677850484848, -0.8677297830581665, -0.3643014430999756, -0.8974437117576599, 0.46389707922935486, -0.9526033997535706, -1.2624855041503906, 0.5859020948410034, 4.143845558166504, 0.4187021255493164, -0.670410692691803, 1.177797555923462, -0.04766249284148216, -0.8411969542503357, 0.9977641701698303, -0.15204045176506042, -0.103089839220047, 0.8971859812736511, 0.5863151550292969, -0.6022328734397888, -2.1719541549682617, 0.7327654361724854, 0.16477113962173462, 1.1290215253829956, 0.048603568226099014, -0.33017006516456604, 0.03374890610575676, 0.978975772857666, 1.6281486749649048, 0.4891892671585083, -1.7404260635375977, -0.07012620568275452, -0.7840902209281921, 0.890259325504303, -0.5178777575492859, -0.8404080867767334, -0.15734638273715973, 0.8718822598457336], [-0.25091543793678284, 0.7007227540016174, -1.6342504024505615, 1.3684110641479492, -0.6422242522239685, 0.21551138162612915, -0.7543126940727234, 0.4530892074108124, -0.11233493685722351, 0.1342158019542694, 0.44907429814338684, 0.07741299271583557, 0.9251545667648315, -0.01062457449734211, 0.20211748778820038, 1.935300588607788, -1.8556859493255615, 0.029087521135807037, 1.1259227991104126, 0.6893431544303894, -0.5137678980827332, -0.5074747204780579, -1.0871883630752563, -1.321816325187683, 0.1868066042661667, 0.11362092196941376, 1.1759157180786133, -0.783025324344635, 0.21279162168502808, 0.6469132900238037, 0.4102163016796112, 0.3972158432006836, -0.14597132802009583, -0.4098086655139923, 0.6644402146339417, 0.43339210748672485, 0.9169555902481079, 0.5316488742828369, -0.20375165343284607, 0.11189499497413635, 1.2167532444000244, 0.1818578839302063, -2.1331114768981934, -1.6899152994155884, -1.275181770324707, 0.09651882946491241, -1.6655666828155518, -1.094852089881897, -1.7576264142990112, 0.03524763509631157, -0.23794974386692047, -1.4644473791122437, -0.7704920768737793, -1.500662922859192, -1.9501639604568481, -0.474658727645874, 0.8230282068252563, 0.8515294790267944, -0.8319659233093262, 1.1075643301010132, -0.8560450673103333, -0.4095759391784668, -1.2006274461746216, -0.6266592741012573], [0.8623866438865662, 0.024394948035478592, -0.505763828754425, -1.7495019435882568, -1.011588215827942, -1.2280969619750977, 0.14587417244911194, 1.020424485206604, 1.0849798917770386, -0.9119437336921692, -0.1765010505914688, 0.8199068307876587, -1.3191033601760864, -0.24393178522586823, -0.3621147572994232, -0.7122299075126648, 0.40251147747039795, 0.6522727608680725, 0.982160210609436, 0.03369137644767761, -0.29217496514320374, 1.7224675416946411, 0.8700401782989502, -1.1407684087753296, 1.2303847074508667, -1.0131045579910278, -0.41996949911117554, 0.19905027747154236, 0.29105427861213684, -1.1373100280761719, 1.3797430992126465, 0.7504389882087708, 1.2466850280761719, -0.40656623244285583, 0.6653516888618469, 1.15989351272583, -1.5537033081054688, 0.5108414888381958, 0.8306283950805664, 0.4458051323890686, -1.1285370588302612, -1.794620156288147, -0.3989264667034149, 0.6891451478004456, -0.5291042923927307, 2.750086545944214, 0.16127604246139526, 0.12285175919532776, 1.7911099195480347, 0.6354526281356812, 0.2312706708908081, -0.4520164132118225, 0.45387932658195496, -0.6565747261047363, -0.042746078222990036, 0.23845092952251434, 0.7294516563415527, -0.06497415155172348, 1.5799016952514648, 0.5344370007514954, -0.5142152309417725, -0.2119990885257721, -0.8249253034591675, -2.0888750553131104], [-0.5933658480644226, 0.2523541748523712, 0.18728238344192505, -1.8281670808792114, 0.2966650128364563, 0.048341505229473114, -0.11747667193412781, -1.62105393409729, 0.3535289168357849, -0.418268084526062, -0.1734432578086853, 0.40329644083976746, 0.36422887444496155, 1.3130970001220703, 0.4020724594593048, 1.100488305091858, 0.10444487631320953, 0.9532681703567505, 0.38621747493743896, -0.25633981823921204, 0.05396788567304611, -0.4582808315753937, -0.6817601919174194, 1.0385479927062988, -0.3372229337692261, 0.4900437891483307, -1.5497186183929443, -0.2750180661678314, 1.479654312133789, 0.5276839733123779, 0.8880601525306702, 1.2502938508987427, -0.23777878284454346, -0.6027715802192688, 0.04840053990483284, -0.6844245195388794, -1.0120162963867188, 0.5287173986434937, -0.7472153306007385, 0.8342612385749817, -0.13299240171909332, 0.8086480498313904, 0.7229924201965332, 0.8116807341575623, 0.1320052295923233, -0.7279532551765442, -0.49143338203430176, -2.082610607147217, -0.1992085725069046, 0.8890979886054993, -1.237492561340332, 1.419918179512024, -0.59764564037323, -0.5933899283409119, -0.2835340201854706, -0.2278572916984558, 0.9285134673118591, 0.05646323040127754, 1.345497488975525, -0.2366751730442047, -1.0905277729034424, -0.827517569065094, -1.3179792165756226, 1.0705751180648804], [0.3924422562122345, -0.39200904965400696, 0.6833220720291138, -1.6717242002487183, 0.24269366264343262, -0.9109343886375427, 1.134979248046875, -0.9794479012489319, -1.6312206983566284, -0.213613361120224, 1.3415864706039429, -1.0759108066558838, -0.1688643842935562, -0.36107972264289856, -1.1837904453277588, -0.543197512626648, -0.1640463024377823, -2.457397699356079, -0.043488774448633194, -0.38367998600006104, -0.8175621628761292, -1.3384162187576294, -0.08990583568811417, 0.4640677571296692, -0.12433008849620819, -0.8128581047058105, -1.0365742444992065, 1.8013501167297363, -0.30947765707969666, 0.19473718106746674, 1.2230021953582764, -1.4323546886444092, -1.3120439052581787, 0.03298119083046913, 0.651417076587677, -0.8682621717453003, 0.2133665829896927, 0.12067056447267532, 2.0688369274139404, 0.1950426697731018, 0.12871897220611572, -1.1844409704208374, -0.49928972125053406, -0.8981070518493652, -0.06700317561626434, 0.056039486080408096, -1.8359951972961426, -0.18302752077579498, -0.8822299242019653, 0.48575595021247864, -0.6186350584030151, -1.1471266746520996, -0.9270181655883789, 0.4871390461921692, -1.3990687131881714, -0.17483608424663544, 0.027040619403123856, 0.5773919820785522, -2.0736098289489746, 0.2845757007598877, -0.6310184597969055, -0.7109888195991516, -0.004918227903544903, -0.4577762186527252], [0.5114604234695435, 0.42731356620788574, -0.9637987017631531, -1.742361068725586, -0.4381978511810303, -0.752271294593811, 1.3268681764602661, 0.05181020125746727, -1.5258492231369019, 0.3142978847026825, -0.46746495366096497, -0.16661043465137482, 0.29621800780296326, 0.25577884912490845, -0.12162584066390991, 0.48028871417045593, -0.9956845045089722, 0.2094731628894806, -0.3742176592350006, 0.9541770219802856, -0.24835088849067688, 1.5294427871704102, -0.37679749727249146, -0.034902382642030716, -0.5404111742973328, -0.9718968272209167, 1.8365377187728882, 0.21781101822853088, -1.5329606533050537, 0.5102400779724121, 0.8454644680023193, 0.3584563136100769, 1.2718563079833984, -0.29842668771743774, -0.3791764974594116, -1.095689058303833, -0.10549422353506088, -0.6123562455177307, 1.176147222518921, 0.9195062518119812, -0.2616272568702698, 1.4002809524536133, 0.21631821990013123, -1.1371066570281982, 0.3663293719291687, 0.4500746726989746, 1.2706162929534912, -0.25630101561546326, -2.0875916481018066, 0.10379335284233093, 0.19897149503231049, 1.0494290590286255, -0.060515694320201874, 0.5686964392662048, 0.8978884816169739, -0.33970633149147034, 0.669511616230011, 0.9126712083816528, 1.3899635076522827, -0.1370682567358017, -0.6767026782035828, 2.092231035232544, -0.9636443257331848, -0.5685116052627563], [-1.356981873512268, 1.147257685661316, 1.809079885482788, -0.5680685043334961, -0.23827548325061798, -1.983252763748169, 0.2701025605201721, 0.27477821707725525, 0.7244473695755005, -0.24592767655849457, -0.8898780345916748, 0.9891347885131836, -1.718994140625, -0.9972221255302429, -0.6251434683799744, 0.8485116362571716, 1.8901596069335938, 0.10018673539161682, 1.0658341646194458, -2.1620125770568848, -1.1320627927780151, -0.8343789577484131, 1.6752065420150757, -0.0072632464580237865, -0.22077970206737518, -1.3627008199691772, 0.1270071566104889, -0.5998610854148865, 0.4875955283641815, -0.6032058000564575, -2.1046929359436035, 1.0689399242401123, -0.14457830786705017, 0.15073956549167633, 0.036445945501327515, 1.8485360145568848, 0.5199835300445557, 0.4505614638328552, -1.0204657316207886, -0.28351467847824097, 1.5004111528396606, -0.9445326924324036, -0.015205265954136848, 0.2638669013977051, -0.185677632689476, -2.0628087520599365, -0.26366573572158813, 1.0657660961151123, -0.36169853806495667, 0.14576727151870728, 1.0104467868804932, -1.2409517765045166, 1.532050371170044, -0.3356010913848877, 0.6529464721679688, 2.486699104309082, -0.6678339242935181, 1.8199706077575684, -0.8784898519515991, -0.2683669626712799, -0.28563374280929565, 1.9322161674499512, -0.10920294374227524, 0.8146854043006897], [1.440071702003479, 1.5094890594482422, 0.21276184916496277, 0.5086039900779724, 0.5393548607826233, -0.8811779618263245, 0.6642822027206421, 0.26869475841522217, -0.45358461141586304, 0.5546655654907227, -0.32367685437202454, 0.9756520390510559, 0.350422203540802, -0.38835981488227844, -0.3058701157569885, -0.0027553748805075884, -1.0028496980667114, 0.18538394570350647, -0.38191312551498413, -0.05024482309818268, -2.0619585514068604, 0.8795249462127686, -0.47912997007369995, -0.35224106907844543, 0.5288568735122681, 1.6411700248718262, 0.8956701159477234, 1.877355933189392, -0.3692700266838074, 1.008133888244629, 1.2574061155319214, 0.2052123248577118, -0.7488580942153931, 1.1642699241638184, 0.9859167337417603, 1.0663594007492065, -0.17170628905296326, -1.9652695655822754, -0.3287186920642853, -0.2738680839538574, 0.07191793620586395, -0.1449958086013794, -0.708531379699707, 0.2841120958328247, -1.384528398513794, 1.610615849494934, 1.4915084838867188, -0.7418220043182373, 1.4028548002243042, -0.018495775759220123, 0.3554159998893738, -0.21182669699192047, -0.3734583258628845, 0.09251454472541809, -0.6129039525985718, 1.0110447406768799, -0.6391050815582275, 0.42987650632858276, -0.5672881603240967, 0.8546117544174194, 2.066114664077759, -0.16840779781341553, -1.7808881998062134, 0.446095734834671], [-0.3647538721561432, 1.470604658126831, -2.3271241188049316, -0.1570383459329605, 0.14947715401649475, -0.4956772029399872, 1.579932451248169, 1.1511640548706055, 1.356119155883789, -1.1164085865020752, 0.04416891932487488, 0.4723198711872101, 1.6157703399658203, -2.1895105838775635, -1.2872097492218018, -0.9176288843154907, 0.9512031674385071, 1.9314296245574951, 1.3085849285125732, -0.15116780996322632, -0.1294693946838379, 0.3998180031776428, 0.32737162709236145, 0.4439457654953003, 2.2582433223724365, 1.2392523288726807, 0.8238360285758972, -2.1008853912353516, -0.20710588991641998, 0.4738052487373352, 0.4132201373577118, 1.20634925365448, -0.8581252098083496, -2.0164542198181152, 0.9639945030212402, -1.9800082445144653, -0.5678135752677917, -1.854873538017273, -0.906843364238739, 0.8087648749351501, 0.7890225052833557, -1.2644051313400269, 0.4223272204399109, 0.41798755526542664, -0.056508421897888184, 0.4585607647895813, -0.5912859439849854, -0.3786715567111969, -0.6867419481277466, 0.20734205842018127, 0.5177478194236755, 0.6914489269256592, -0.27119913697242737, 0.1637733429670334, 1.0187700986862183, -0.9785728454589844, -0.2410988211631775, 0.03730820119380951, 0.3000977337360382, -0.017795350402593613, 0.305702269077301, 0.4269467890262604, 0.48264622688293457, 0.6870076656341553], [-0.06944761425256729, -1.3294458389282227, 1.0021605491638184, -1.5813089609146118, 0.4437485933303833, -0.8065789937973022, -0.5731531381607056, 1.2606639862060547, -0.687378466129303, -0.12507064640522003, -0.14963717758655548, -1.038527250289917, 1.4023194313049316, -0.9719309210777283, -0.5722951292991638, 1.8505544662475586, 1.0134809017181396, -0.5448234677314758, -0.6584895253181458, -1.9340046644210815, -1.1958554983139038, -0.6974249482154846, -0.16435764729976654, 0.19956697523593903, -1.3126133680343628, 2.5242857933044434, 0.44759437441825867, 1.0365393161773682, -0.26225224137306213, -0.7681255340576172, 0.6842110753059387, -1.2812432050704956, 0.10841266810894012, 1.2635200023651123, -1.2681137323379517, -1.1726795434951782, 0.6837517023086548, -0.9621706604957581, 1.8471400737762451, -0.6808139085769653, 1.4549084901809692, -0.7176935076713562, -0.732900857925415, 0.6693004965782166, -1.237432599067688, 1.5344759225845337, -0.7978783845901489, -0.7353569269180298, -0.34832751750946045, 0.14906588196754456, -0.011314517818391323, 0.3001178205013275, 0.34112000465393066, 0.4477962553501129, 0.09479454904794693, 0.2695195972919464, -0.4315376877784729, 0.43373239040374756, -1.5413172245025635, 1.2639212608337402, -0.23223064839839935, 0.6506581902503967, -0.9200245141983032, -0.5970609188079834], [-0.13769544661045074, -1.0405211448669434, -2.2702462673187256, 0.22608934342861176, -2.0698750019073486, 0.6917504072189331, 0.4665583372116089, 0.4714576005935669, -0.7269158959388733, -0.6601399183273315, -0.19960449635982513, 1.5623127222061157, -1.562987208366394, 0.5763434767723083, 1.686835765838623, 1.3221685886383057, -0.42854663729667664, 0.052288033068180084, 0.674379825592041, -0.4468057155609131, -1.5053033828735352, 0.47403860092163086, 0.3550776243209839, 1.4296793937683105, 0.5236067175865173, 1.3710805177688599, 0.15212149918079376, 0.6393541693687439, -1.2811551094055176, 0.1724247932434082, -0.8768611550331116, -2.2351365089416504, -0.45278215408325195, -0.7677515745162964, -1.1670808792114258, 1.1882190704345703, -0.15199416875839233, 1.5549029111862183, -0.1948164850473404, -1.4974777698516846, -0.208848774433136, -0.5878845453262329, -0.332041472196579, -2.160616636276245, 1.1914622783660889, 0.2906542718410492, 0.564085066318512, 0.5576728582382202, 0.8261235952377319, -0.5320163369178772, -1.024139165878296, 0.04169969633221626, 1.6763198375701904, -1.1906253099441528, -0.6701482534408569, -0.07444291561841965, 1.0421642065048218, 0.5421474575996399, -1.1534178256988525, 0.1848042756319046, 0.5625481009483337, -0.11491180211305618, 0.43779224157333374, -0.3591073751449585], [1.044498085975647, -0.2004820555448532, 1.1761071681976318, -0.5175449848175049, 1.0618129968643188, -1.9332287311553955, 1.1530102491378784, -1.0715574026107788, -0.49544766545295715, 0.8703296184539795, 1.0852948427200317, -1.917840600013733, -1.3798112869262695, 0.4325830042362213, 0.7470841407775879, -0.23538440465927124, -1.7009726762771606, -0.6414644122123718, 1.278244137763977, -0.49286088347435, 0.904293954372406, -0.19547319412231445, 1.1193945407867432, -1.4781286716461182, -0.327223002910614, -2.0316367149353027, -1.327526569366455, 0.5343643426895142, -0.07257799804210663, 0.20992422103881836, -0.1637738049030304, -0.6742499470710754, 0.43453964591026306, -0.8328973650932312, 0.7092116475105286, -0.5930207967758179, -0.22270113229751587, -0.2509141266345978, 0.7334370017051697, 0.8100029826164246, -0.8222365379333496, -0.012999730184674263, 0.02366730198264122, 1.6248575448989868, -0.7571730017662048, 0.29080086946487427, 0.4044544994831085, 0.8478586077690125, 0.27000534534454346, 0.5284578204154968, 1.6596980094909668, 0.741291880607605, -1.2460275888442993, -0.7212238311767578, 0.11935222148895264, -0.31568485498428345, -2.274589776992798, -1.303095817565918, -0.7751824855804443, -1.0497883558273315, -0.993488073348999, -0.23141564428806305, -2.449976921081543, -0.15964990854263306], [-0.06804489344358444, 0.8238851428031921, -0.04866112768650055, -0.6966419219970703, 0.947384774684906, -1.122353434562683, 1.014575481414795, -1.7176685333251953, 1.6653836965560913, 0.6177353858947754, 0.3877296447753906, -1.1546523571014404, -2.726156234741211, -0.6854039430618286, -1.469710111618042, -0.4753965139389038, 1.929352045059204, 0.28701043128967285, 2.039640188217163, 2.0136947631835938, -1.2159521579742432, -0.5145376920700073, 0.5084845423698425, -1.9163936376571655, 0.6206823587417603, 1.8498377799987793, 0.7436978816986084, 0.05473653972148895, 0.9626070857048035, -1.0084072351455688, 0.6303512454032898, 0.06568305194377899, -1.1534870862960815, 0.8942793011665344, -0.5930849313735962, 0.6217473745346069, -0.24052612483501434, -0.7400796413421631, 0.25210487842559814, 0.5688700079917908, -1.2355337142944336, -0.20427733659744263, 0.040495261549949646, -0.3545263409614563, 1.912870168685913, -0.29170748591423035, 0.3630217909812927, 0.7770538926124573, 0.07548472285270691, -0.26893290877342224, 1.307288408279419, -2.1234724521636963, -1.2916768789291382, 2.022839307785034, -1.2096797227859497, 0.9388498067855835, 0.8266826868057251, 1.3444477319717407, -2.065884590148926, -1.9438003301620483, 1.295310616493225, 0.4915589690208435, -1.002571702003479, -0.7636500597000122], [0.7562404274940491, 0.2013593316078186, -1.1033681631088257, -1.5990101099014282, 1.1882206201553345, 0.4380200505256653, -2.532850980758667, -0.8801476955413818, 2.464057683944702, 0.8791606426239014, 0.9424561858177185, 2.3521430492401123, -1.0859370231628418, -0.11664552986621857, 0.6419233083724976, 0.17343014478683472, 0.4477073550224304, -0.08099991828203201, -2.070457935333252, -0.08181248605251312, 1.411583662033081, -0.1707381159067154, 0.4344448149204254, -0.7154069542884827, 0.5421921014785767, 0.3891269564628601, -0.206198051571846, -0.2823293209075928, 0.3297569453716278, -0.3880273401737213, 0.723091185092926, -0.6888465881347656, -0.13716574013233185, 1.659386157989502, 0.42414242029190063, 0.30800604820251465, -0.757203996181488, 0.31409692764282227, 0.5238662958145142, -0.507064700126648, -0.04102995991706848, -1.036380648612976, 0.45336997509002686, -0.9705603718757629, -0.3898668587207794, 0.4138420522212982, -1.8818532228469849, -0.41680052876472473, -0.9088108539581299, 0.5052685737609863, -0.24596905708312988, 0.5394302010536194, 0.9577672481536865, -0.9731282591819763, -1.182104468345642, -0.5583438873291016, -0.19241635501384735, 1.0055606365203857, 0.31315529346466064, -0.32957181334495544, 0.35164058208465576, 0.3438636362552643, -0.8945603370666504, 0.8009557127952576], [0.1305401772260666, 0.6913918256759644, -0.10778431594371796, -0.9721446633338928, 0.34964314103126526, -1.1960614919662476, 0.13968302309513092, -1.1182769536972046, -0.10163410753011703, 2.0996077060699463, 0.7511391043663025, 2.0379104614257812, -0.25318291783332825, 0.423696368932724, 0.9966456890106201, 1.212749719619751, -0.8219308257102966, 1.3563175201416016, -0.6469385623931885, -1.1076793670654297, -0.4412451386451721, 1.6113173961639404, -1.2301839590072632, 1.0674383640289307, 1.4795271158218384, -1.0105887651443481, 0.8720544576644897, -0.8292829394340515, 1.7145497798919678, -1.6219415664672852, -0.5463306903839111, -2.7145748138427734, -0.3789428472518921, 0.4528225362300873, 1.5805938243865967, -1.4989943504333496, 1.7029398679733276, -0.7364606857299805, 0.09750354290008545, -0.8201099634170532, 0.7207410931587219, 0.01554677914828062, -0.26781710982322693, -1.1903941631317139, 0.40798142552375793, 0.12912505865097046, 0.32527029514312744, 0.31005552411079407, 1.1475085020065308, -0.48294299840927124, 0.1377047598361969, -0.9540889859199524, -1.5681639909744263, 0.24602819979190826, 0.19762422144412994, -0.05675981938838959, -0.38582128286361694, 0.6493348479270935, -0.2961694002151489, -0.05399399250745773, 1.1974323987960815, -1.2563050985336304, 0.7155839204788208, -0.5251745581626892], [0.9401577115058899, 0.9055402874946594, -2.0039496421813965, -0.04704373702406883, 0.09867274761199951, 0.2964267134666443, 0.97837233543396, 0.8673037886619568, -1.355880856513977, -1.5290840864181519, 0.6502088308334351, -0.16126608848571777, -0.8283066153526306, 1.1596291065216064, -1.1638976335525513, -0.9332671165466309, 1.3134897947311401, -0.2739783227443695, -0.5980507135391235, 0.7820138335227966, -1.9848378896713257, -0.31445223093032837, 0.13593338429927826, -0.36001718044281006, -1.6731562614440918, -1.0393356084823608, 0.7045361995697021, 1.6952104568481445, -2.1950321197509766, -0.9583700895309448, -0.04858648404479027, 0.12224412709474564, -0.04312858358025551, -0.2773967385292053, 1.0896461009979248, 1.0839800834655762, 1.2297378778457642, 0.20704029500484467, 0.1710558980703354, -0.22671601176261902, 0.07701192796230316, -1.761705756187439, -0.5667039155960083, 0.0480317696928978, -0.03630590811371803, 1.1906548738479614, 0.5274113416671753, 2.0659804344177246, -1.8779566287994385, -0.9563544392585754, 1.0766329765319824, -0.3223806321620941, -1.024914264678955, -1.0661171674728394, -0.5604815483093262, -0.10769831389188766, -1.1520966291427612, 2.017536163330078, 1.22443425655365, -1.4856692552566528, 0.9228864908218384, -0.9926050305366516, -0.14504693448543549, 1.3172534704208374], [-0.44967377185821533, -0.2541857659816742, -0.35710757970809937, 1.276364803314209, 0.8947771191596985, -0.016045721247792244, -0.7556622624397278, 1.525537133216858, -0.28659284114837646, 0.8830321431159973, -0.8698704242706299, 0.04397334158420563, -0.5837404131889343, 1.6346805095672607, -0.7717040777206421, 0.8054523468017578, 0.11742060631513596, -0.5923808217048645, -2.143904685974121, 1.342862844467163, 1.3550726175308228, 2.0296783447265625, 1.5364679098129272, -0.5542284846305847, -0.8560562133789062, 0.09233084321022034, 0.335459440946579, 1.101983904838562, -0.5361701846122742, 0.974615216255188, 0.8271746635437012, 1.6801886558532715, -1.044385552406311, -0.03701014444231987, -0.10467308014631271, 0.7867313027381897, -1.231740951538086, 0.7260598540306091, 0.6784956455230713, -0.7135549187660217, -0.33552759885787964, -0.15281900763511658, -0.09167693555355072, 2.3077666759490967, 2.18331241607666, -0.291805624961853, -0.6248956918716431, -1.6772935390472412, -1.0526639223098755, 0.44428348541259766, -0.3189382255077362, -2.66009259223938, -0.36666664481163025, -0.5579973459243774, 0.7052154541015625, 0.4011267125606537, -1.0145587921142578, 0.33307188749313354, 0.012257984839379787, -0.9441841840744019, 1.396060824394226, 0.1627175360918045, -0.87882399559021, -0.3204552233219147], [-0.12264221161603928, -0.5557852387428284, 0.8067983984947205, -0.037091214209795, 0.2952418625354767, -0.22441434860229492, -1.358355164527893, -0.5501745343208313, 0.7599584460258484, 0.5624116659164429, 0.038630496710538864, -0.2164829820394516, 0.6700991988182068, 0.4806438684463501, 0.846000075340271, 0.6271442174911499, -0.10938506573438644, -0.3870126008987427, -0.9349647760391235, -1.6021437644958496, -1.3586037158966064, 2.1840133666992188, -0.03884437307715416, -0.9483101963996887, 0.7593802213668823, -0.7021693587303162, 1.704767107963562, 2.451833724975586, -0.1654987931251526, 1.2360538244247437, 1.5538190603256226, -0.5300301909446716, -0.4239879250526428, 0.867658257484436, -0.8018521666526794, -0.793591320514679, 0.4984515309333801, 0.32957616448402405, -1.2530359029769897, -1.597887635231018, -1.455795168876648, 0.47404560446739197, -0.6198286414146423, -1.4497350454330444, -1.5286681652069092, 1.0292903184890747, 1.1704685688018799, 1.4421412944793701, -0.6100898385047913, 0.4905310571193695, -0.6123223900794983, 1.0679967403411865, -0.05943533033132553, 1.3312355279922485, 1.177106261253357, 0.5149883031845093, 0.4100004732608795, -0.6074542999267578, 0.09454279392957687, 0.38668879866600037, -0.9905192852020264, -0.37907055020332336, 0.24254027009010315, -0.9419506192207336], [0.3485517203807831, 0.5751526355743408, 0.5327650904655457, -1.5965638160705566, -0.1306559443473816, -0.9511775374412537, -0.23828847706317902, 0.6637169122695923, 0.23255933821201324, 0.21119730174541473, 0.5360081791877747, -1.1468452215194702, -0.741468071937561, -0.04747576266527176, -2.1909737586975098, 0.9099515080451965, 0.3240334093570709, -0.6139925122261047, 0.39558228850364685, -2.213754653930664, -0.06342576444149017, -0.1986554116010666, 2.0897927284240723, -0.15433605015277863, -0.9103822708129883, -0.943217933177948, 0.7573249340057373, 0.31385907530784607, -0.7816933393478394, 0.12077977508306503, -1.893615961074829, 1.4405714273452759, 0.25207164883613586, 0.07516966015100479, -0.21075962483882904, -2.708071231842041, -0.7479324340820312, -0.8943771123886108, -0.6631339192390442, -0.16574618220329285, -0.8877270817756653, 0.8565530776977539, -0.6039801836013794, 0.24072414636611938, -0.0012686492409557104, 1.3217741250991821, -0.40062254667282104, 0.5026121139526367, 1.1128467321395874, 0.803382396697998, -0.03577204793691635, -1.6565537452697754, 1.1639491319656372, 1.3512563705444336, 0.24543143808841705, -1.1180208921432495, 0.23354701697826385, 0.39096033573150635, 1.8141430616378784, 0.713359534740448, -0.8975367546081543, -0.4066488444805145, -0.3754437565803528, 2.2961513996124268], [-0.8870883584022522, 2.571622610092163, -0.20217961072921753, -0.8379698991775513, -0.535065233707428, 0.49504318833351135, 0.6755589842796326, 0.4035547077655792, -0.5809392929077148, -0.13449035584926605, 0.3821864724159241, 0.6802245378494263, 0.581503689289093, -1.3846549987792969, 0.914506196975708, 0.5990946888923645, 0.4408872127532959, 0.012610224075615406, 0.3124898374080658, 1.1948671340942383, -0.5771691203117371, 1.0223606824874878, -1.1190686225891113, -1.313739538192749, -0.6821131110191345, -1.2796951532363892, -1.008021354675293, -0.8161234259605408, -0.33304154872894287, -0.9819914102554321, 0.47091835737228394, 0.7619603872299194, -0.9935807585716248, 0.4572257995605469, 0.9614257216453552, 0.831882119178772, -1.4602675437927246, -0.39829275012016296, 0.41804012656211853, -1.1280500888824463, -0.1367059051990509, 0.42216548323631287, -0.8522624969482422, 0.12581712007522583, -0.5932633876800537, 0.060279715806245804, 0.8019864559173584, 0.6253642439842224, -1.103359580039978, -0.17118555307388306, 0.02885284088551998, 0.7957586646080017, -0.8021396994590759, -1.0615437030792236, 0.042336560785770416, 0.4729965031147003, 0.4477385878562927, 0.15837687253952026, -0.09541326016187668, -1.1664552688598633, 0.3654196262359619, 0.6273915767669678, -0.05397272855043411, 0.49446967244148254], [-0.08434019982814789, 0.30990520119667053, -0.2904120087623596, 0.14005932211875916, -0.2912183701992035, 0.9666820764541626, 0.42995864152908325, 1.0601329803466797, -0.5909063220024109, -1.0961745977401733, 0.04972048103809357, -0.3744618594646454, -0.012912150472402573, 0.3925698697566986, -0.6075296998023987, 0.3399869501590729, -1.11910879611969, 1.2679048776626587, -1.2819010019302368, -0.36783796548843384, -1.2236262559890747, 0.04678640514612198, -1.7319748401641846, 0.000511312682647258, -1.1923370361328125, 0.7066403031349182, 0.5554348826408386, 1.0071486234664917, 0.8796941041946411, -0.7490702867507935, 0.34914690256118774, 0.10778743773698807, -1.3942296504974365, -0.11560415476560593, 1.4426277875900269, -0.42402383685112, -1.064621925354004, -1.321280837059021, -0.9336378574371338, 1.1513115167617798, 0.9165151715278625, -0.5074214339256287, 1.1198033094406128, 0.9091309309005737, -0.013565418310463428, -0.10491583496332169, -0.6055974364280701, 1.46098792552948, -1.2177774906158447, 0.437617689371109, -0.7162768840789795, 1.9638593196868896, -1.478494644165039, 0.015733258798718452, 0.7318292856216431, -1.0311042070388794, -0.6368950009346008, -0.5415176153182983, -0.8332317471504211, 1.1873745918273926, 0.19757992029190063, -0.47201529145240784, -0.850334107875824, 0.05053430795669556], [0.6854336857795715, 0.5219769477844238, -1.674965739250183, 1.3255563974380493, 0.5738548636436462, -1.9299099445343018, -0.9268274903297424, -1.675859808921814, 0.42341122031211853, -0.01736493781208992, 1.3017016649246216, -0.6191791296005249, 0.09984075278043747, -0.00528751639649272, 0.8841482996940613, 0.2890574336051941, -0.5952046513557434, 1.0321285724639893, -0.4984396994113922, -0.2699003219604492, 0.8688445687294006, -0.027786586433649063, 0.9145646095275879, -0.17793411016464233, -0.13665202260017395, 0.3717799484729767, -1.2446470260620117, -0.1578635722398758, 0.6777700781822205, -1.1847078800201416, 2.365687847137451, 0.909861147403717, 1.3186652660369873, 0.47622042894363403, -0.6431531310081482, 0.6527441143989563, 0.5642470121383667, -0.9633089303970337, 0.663198709487915, -0.2393968552350998, -2.2666831016540527, 0.1663934290409088, -1.0705806016921997, -1.3040294647216797, -0.9121306538581848, 0.4162582755088806, 0.37193381786346436, 1.1846075057983398, 1.5819894075393677, -1.3817936182022095, 1.3043856620788574, 0.7018330693244934, -1.1008447408676147, -0.7593690752983093, 0.7603339552879333, -0.3215469419956207, 0.7035519480705261, 0.678128182888031, -0.4704364240169525, -0.9714690446853638, -1.3664125204086304, -1.5507351160049438, 0.21840709447860718, -1.207067608833313], [0.03944084420800209, -0.751359224319458, 1.066009759902954, -1.0861337184906006, -0.3204335868358612, 1.2670235633850098, 0.7223554849624634, -1.7380542755126953, -2.0005943775177, -0.03111167997121811, -0.9233139157295227, 1.3564074039459229, -0.7365478873252869, -0.28406015038490295, -0.7248764634132385, 0.7754009962081909, 2.5331368446350098, -3.1808464527130127, -1.9587875604629517, -1.2011826038360596, -0.33251330256462097, -0.8758717179298401, 0.31754937767982483, 0.1464097648859024, -1.4610261917114258, -0.8432692885398865, -1.408364176750183, 1.1091828346252441, 0.09377944469451904, -1.8132431507110596, -1.0841859579086304, -0.09796926379203796, -0.9834814071655273, 0.09255530685186386, 0.1257963478565216, -0.05588993430137634, 0.40774160623550415, 1.855830192565918, -1.6114178895950317, -2.0092787742614746, -1.142441987991333, -1.4297109842300415, 0.2934001386165619, -0.007463354151695967, -0.5793821215629578, 0.8718782663345337, 0.6099307537078857, 1.4448922872543335, -0.2404901087284088, -0.3248208463191986, 1.6419821977615356, 0.9928192496299744, -0.3594535291194916, 1.0908963680267334, -0.6875938177108765, -0.28607526421546936, -0.40149322152137756, 1.6026242971420288, -1.278355360031128, 0.6061749458312988, -0.7905983328819275, -0.9182673096656799, -1.1108100414276123, 0.04710337892174721], [-0.9595980048179626, 0.15934570133686066, 1.1787834167480469, 0.12035065144300461, 0.8249331116676331, -0.3097015917301178, -1.2115999460220337, 0.6248543858528137, 0.7553857564926147, -0.8123082518577576, 2.757330894470215, 2.1716980934143066, 0.7619711756706238, 0.7901975512504578, 0.2486806958913803, -0.6618313193321228, 0.947894811630249, -1.2517096996307373, 0.346175879240036, 1.8412628173828125, 1.5790761709213257, -0.30683404207229614, -2.7869958877563477, 0.6159722208976746, -0.164193257689476, 0.7930393218994141, -0.8668335676193237, -0.5722739696502686, 0.39564862847328186, 1.1195131540298462, -0.6334717869758606, -0.14217893779277802, -0.22482338547706604, 0.005771511700004339, -0.3195735514163971, -0.008838322013616562, -1.3436745405197144, -0.5941056609153748, -0.4181373119354248, 2.486814260482788, 0.15453341603279114, -0.5765241980552673, 0.5370314121246338, -0.9601201415061951, 0.6885087490081787, -0.07156231999397278, 0.4600766897201538, 0.3795834183692932, -0.641044020652771, 1.4586915969848633, -1.6325087547302246, 1.7704113721847534, 1.2901148796081543, -0.35108473896980286, -0.2640259265899658, -2.491337299346924, -0.10108869522809982, 0.09525347501039505, 0.7049486637115479, 0.01332777924835682, 0.6969851851463318, -0.03860771656036377, 1.1730008125305176, 0.22332310676574707], [1.225251317024231, -0.44357579946517944, 0.12823759019374847, -1.0192991495132446, -0.9445176720619202, 1.1317343711853027, -0.26085755228996277, -0.842530369758606, 0.6143395900726318, 0.619493842124939, 0.38079971075057983, 1.8493050336837769, 1.603711724281311, 0.5054365992546082, -0.3831769526004791, -0.5598172545433044, -0.7590554356575012, 0.9839721322059631, 0.7032641172409058, -0.7014425992965698, -0.5245925784111023, 0.08994460105895996, 1.1655176877975464, 1.5200567245483398, 2.3425850868225098, -0.39476728439331055, -1.967226266860962, 0.678015410900116, -0.20333395898342133, 0.051141202449798584, 0.17003141343593597, -0.2987819314002991, 1.04010808467865, -0.2251780927181244, 0.8305900692939758, -1.3994590044021606, 2.12837290763855, 1.0113461017608643, -0.9933016896247864, 1.5604207515716553, -0.11874398589134216, -0.5647154450416565, 0.6731595396995544, 0.40699630975723267, -0.2851807773113251, -0.08933071047067642, -1.8060487508773804, -1.437218427658081, -1.6004586219787598, -0.1583269089460373, -0.3545028567314148, 0.9950783848762512, 1.2631222009658813, -0.3555557131767273, 0.3409954905509949, -1.4624245166778564, -0.07994817197322845, -0.31020158529281616, 0.09765072911977768, 1.3513264656066895, 0.8522210717201233, -2.1460866928100586, 0.3749067187309265, 0.34052926301956177], [-1.4415256977081299, -1.942578911781311, 1.2553473711013794, -0.04690096527338028, 0.5097717046737671, -0.7165480256080627, 0.07780835032463074, -0.33821848034858704, -1.6993969678878784, -0.4371964633464813, 0.1810544729232788, 0.5206213593482971, 0.7816181182861328, -0.9111523628234863, -1.955027461051941, -1.2856038808822632, -0.17991195619106293, -0.2686128616333008, 0.0027201990596950054, -1.0879889726638794, 0.44834738969802856, -0.6762056946754456, 1.417081356048584, 0.07843485474586487, 0.4534648656845093, 1.1042629480361938, -0.782061755657196, 0.250575453042984, -1.1244423389434814, 0.7927849292755127, 0.7152530550956726, 0.7586953043937683, 0.004542882088571787, -0.025308843702077866, -0.5256783962249756, -1.4436650276184082, -0.15978340804576874, 1.5473113059997559, 0.7973432540893555, -1.0173348188400269, 0.6353409886360168, -0.20728926360607147, 0.1003960520029068, 1.0574030876159668, -0.475024938583374, 0.26639389991760254, -1.4826205968856812, -1.3729829788208008, -1.4125394821166992, -0.10059674829244614, 0.6380593180656433, 0.45574209094047546, 1.4004528522491455, 2.4775359630584717, -0.7383476495742798, 0.5351001620292664, 1.6000078916549683, -0.5763282775878906, 0.6733543276786804, -0.534652829170227, -0.7181861996650696, 1.618587613105774, -1.3661766052246094, -0.7180097699165344], [1.2054383754730225, -0.17255811393260956, 1.3870339393615723, -0.9363082647323608, -0.6674808263778687, -1.0590099096298218, -0.5557559132575989, -1.8021527528762817, -0.6033353805541992, 0.4942128360271454, -0.9118379354476929, -0.7055889368057251, -0.4015226364135742, 1.9172025918960571, 0.49700525403022766, 0.03676038980484009, 0.8071174621582031, 0.22034215927124023, 0.011860419996082783, -1.317044734954834, -2.661868095397949, -0.4661214053630829, 2.1421332359313965, 0.9690895676612854, -0.6260873079299927, 0.6726824045181274, -0.5342661142349243, -0.88059002161026, -0.9362015724182129, 0.3221699893474579, -1.5159926414489746, 1.4422264099121094, -0.999005138874054, -1.134208083152771, 0.20983931422233582, 0.8368119597434998, 0.5075522661209106, -0.44333416223526, 0.5850491523742676, -2.089792490005493, -0.7216154336929321, -0.5324622392654419, -1.3617910146713257, -0.6609121561050415, -0.3064036965370178, -2.302516460418701, 1.0064938068389893, 1.6574290990829468, -0.5773780941963196, 0.3054344356060028, -2.1943118572235107, 0.5034676790237427, -1.1097244024276733, 0.7400501370429993, 0.17548511922359467, 0.04495730623602867, -0.9809240698814392, -0.22158309817314148, 0.2723095417022705, -1.4231616258621216, -0.9921471476554871, -1.3666179180145264, 0.9983006119728088, -0.5704917907714844], [-0.15747828781604767, 0.36346206068992615, 0.7182012796401978, 0.29568690061569214, 0.6015632152557373, -0.5589761734008789, -1.2128194570541382, -0.7500116229057312, -2.0059521198272705, 1.2213112115859985, -0.1698002815246582, -0.6520894765853882, -0.6098285913467407, -1.243550181388855, -0.5908864736557007, 0.10886016488075256, 1.4834620952606201, 0.7433258891105652, 0.521558940410614, 0.9175884127616882, 1.2627753019332886, 0.1123301088809967, -0.15407373011112213, 0.46084848046302795, 0.47784683108329773, -0.652342677116394, 0.691401481628418, -0.992240309715271, 1.9268687963485718, -1.5724480152130127, -0.33017629384994507, 0.26642024517059326, -2.123156785964966, -0.7195096611976624, 2.7006678581237793, -0.2706514596939087, 1.134163498878479, -0.2924805283546448, 0.6319248080253601, 0.09103913605213165, 1.8719818592071533, 1.6005899906158447, 0.17569594085216522, 0.44004276394844055, 1.0905208587646484, 0.4169701635837555, 2.608691930770874, -0.2813611626625061, 1.5504515171051025, 1.5274710655212402, -1.5438120365142822, -0.9207205772399902, 0.23589394986629486, 0.4365415573120117, 0.8245388865470886, 1.2778041362762451, -1.7110415697097778, 0.5825170278549194, 0.2244889736175537, -0.924595832824707, 0.07413303852081299, -0.7923028469085693, -1.1764410734176636, 0.7597101330757141], [-0.34513232111930847, -0.11227405071258545, 1.8772460222244263, 0.5907115340232849, 0.6044265627861023, 0.6839077472686768, -0.3038874566555023, -1.1517890691757202, -0.7084789276123047, 0.2503264546394348, 0.9115725755691528, 0.2615698277950287, 1.9957408905029297, 0.32213446497917175, -1.9173928499221802, 0.36609354615211487, 0.3173518776893616, 0.5483877062797546, 0.6359316110610962, -0.09314510226249695, 0.6962313055992126, -0.9080220460891724, -0.3185702860355377, 0.4756948947906494, 2.7568278312683105, 1.187807321548462, 0.9338194131851196, 1.142185091972351, 1.3406004905700684, -0.20524005591869354, 0.5717132091522217, -0.28706416487693787, -0.32943689823150635, 1.4430290460586548, 1.1572169065475464, 0.7924946546554565, 1.6770339012145996, 0.364214152097702, -0.11387467384338379, 0.10569639503955841, 1.0875054597854614, -0.6057817935943604, 0.49092620611190796, -1.5434479713439941, 0.007883102633059025, -0.07955056428909302, -0.2685861587524414, -0.054140057414770126, 0.11509653925895691, 0.5464805364608765, -0.4861232042312622, 1.6564064025878906, 0.2677779793739319, -0.17109820246696472, -1.496866226196289, 1.163486361503601, -0.9269570112228394, -1.542317271232605, -1.1102964878082275, -0.9726666212081909, 0.2018875777721405, 1.3483413457870483, -0.21573396027088165, 0.5361045598983765], [-1.7485313415527344, -0.214466854929924, -0.5342776775360107, -0.09352283924818039, -0.9891048073768616, -1.1899526119232178, -0.4939662516117096, 2.0394771099090576, -0.1806132048368454, 1.1145603656768799, 1.8570566177368164, -0.03851940110325813, 0.4279184639453888, 1.4927512407302856, -1.178762674331665, 0.48322024941444397, -1.5945998430252075, 0.18127329647541046, -0.15058128535747528, 0.5843046307563782, -1.5370972156524658, -0.9136011004447937, 0.3418901860713959, 1.4207371473312378, 0.19780589640140533, 0.7033371925354004, 1.2317161560058594, 0.9272493124008179, 0.3815128207206726, 0.17081613838672638, -0.5919337272644043, 0.3167429268360138, -1.4445092678070068, 0.07141103595495224, 0.9251136183738708, 0.3027476668357849, 0.14060580730438232, 0.13508069515228271, -0.6333882212638855, -0.9049552083015442, -0.11138548702001572, 2.4947304725646973, -0.6538504362106323, -0.4178968369960785, -0.4162033498287201, -0.4552849233150482, 0.4662933945655823, 1.039156198501587, 0.08782761543989182, -1.5406616926193237, 2.05253267288208, -0.0506305992603302, -1.3545596599578857, 0.6562639474868774, -0.047817960381507874, -0.7965571284294128, -0.12780576944351196, 1.0351659059524536, 0.32956215739250183, 0.6653121709823608, -0.8486655950546265, -0.6296631693840027, 0.26417502760887146, -0.9075775742530823], [-0.8954700827598572, 1.3259680271148682, -2.0895094871520996, -0.5122019052505493, 2.475151777267456, 0.027010271325707436, 0.3372875452041626, 0.24684765934944153, -0.1473342329263687, 0.5885524749755859, 0.6990540623664856, -0.8977377414703369, -0.33566752076148987, 0.5796839594841003, 0.583182692527771, 0.182320699095726, -0.09423934668302536, -1.2003575563430786, -0.6477240324020386, 1.0573650598526, -0.9580678939819336, 0.3793092370033264, 0.941469669342041, 2.3030219078063965, -0.270753413438797, -1.383424997329712, -0.09951025992631912, -1.244144082069397, -1.9080578088760376, -0.6564949154853821, 0.5640336275100708, -1.6905930042266846, 0.7336989045143127, 0.4335445761680603, 0.876934289932251, -0.05030425637960434, 0.837733805179596, -0.2661324143409729, -0.8793269991874695, -0.045780863612890244, 0.6179329752922058, -0.06352660059928894, -0.8552100658416748, -0.6758452653884888, 0.6874042749404907, 0.4482457935810089, -2.130399227142334, -0.012375805526971817, -0.12524060904979706, 0.5557144284248352, 0.9559069871902466, -0.1293882578611374, -0.5087642669677734, 1.1300017833709717, 1.4918545484542847, 0.6121463775634766, -1.5534214973449707, 0.06052382290363312, 0.22959618270397186, 1.2320938110351562, -0.1113799512386322, 0.0019162670942023396, 0.845139741897583, -0.5858387351036072], [-1.1990638971328735, 1.0847804546356201, 0.4282010793685913, -0.05189639702439308, 1.9570379257202148, 0.7653541564941406, -1.0114117860794067, 0.6140869855880737, 0.30214881896972656, -1.655842661857605, 1.023842215538025, -0.24337948858737946, -1.8879539966583252, 0.5606447458267212, 0.3640390634536743, 1.9831899404525757, 0.9811576008796692, 0.008363476023077965, -0.16827622056007385, -0.37822896242141724, -0.6327539086341858, -0.36296138167381287, -0.7264969944953918, -0.13431352376937866, -0.11680618673563004, 0.9984067678451538, 0.17004503309726715, -0.5132269263267517, 0.06309504061937332, 1.1946038007736206, -0.37374410033226013, -2.7753477096557617, 0.2950766086578369, -0.17159400880336761, 0.024935269728302956, -1.1162172555923462, 0.16744638979434967, 0.12282650917768478, 0.5676716566085815, 0.41295790672302246, 0.3406105041503906, -0.5395985245704651, -0.09316100925207138, 0.10834070295095444, -0.44007518887519836, -0.2095087319612503, 0.8103834390640259, -0.7266620397567749, -1.0799987316131592, 1.5317906141281128, -0.32473066449165344, 1.1645666360855103, -0.46738889813423157, 0.07437492907047272, -0.5002274513244629, -1.166538119316101, -0.23523665964603424, -0.6064971685409546, -0.14120008051395416, 0.6585419774055481, 0.2482946515083313, 0.07755398005247116, -1.571103572845459, -0.9138541221618652], [0.3919626474380493, 0.034640856087207794, 1.866473913192749, -0.7377173900604248, -0.6665633320808411, -0.20816649496555328, 1.1170637607574463, -1.062936544418335, -1.7936447858810425, -0.06160416081547737, -1.0667750835418701, 1.176121473312378, -0.11452380567789078, -2.6304376125335693, 0.021152375265955925, -2.6999776363372803, 0.13454627990722656, -0.26466742157936096, -0.6514003872871399, 0.8092271089553833, -0.4005354940891266, -1.2456611394882202, -0.8754402995109558, 0.10900530219078064, 0.415033757686615, -1.3866519927978516, 1.7431172132492065, -0.8552647829055786, -1.6849963665008545, 0.10843705385923386, 0.6149540543556213, 0.4719705581665039, 0.3471856713294983, -2.369976758956909, 1.1090699434280396, 1.5800187587738037, 0.5987759828567505, -1.1503033638000488, -0.77916020154953, 0.3608960807323456, 0.03209839388728142, 0.370533287525177, 0.7584019899368286, -0.0415167473256588, 0.8096083998680115, -0.6807783842086792, 0.510667085647583, 0.3792826533317566, 0.860954761505127, -0.6062898635864258, 0.04649950563907623, -1.4172132015228271, 0.24792549014091492, 0.8511108160018921, -2.328559160232544, -2.3337979316711426, -1.8607275485992432, -0.3962990641593933, -0.31323355436325073, -1.62615168094635, 0.46340423822402954, 1.0644670724868774, 0.6152724027633667, -0.8371936082839966], [1.2135028839111328, 0.3265033960342407, 0.26257947087287903, -0.1511642336845398, 0.03151702508330345, -1.6551120281219482, 0.48811855912208557, 0.24453739821910858, 0.5126672387123108, 1.0578181743621826, -0.129537433385849, 0.9662789702415466, -0.18860267102718353, 0.674688458442688, -1.8082653284072876, 0.8836643695831299, -1.2052741050720215, -0.08222658932209015, -0.3637094795703888, 1.3407272100448608, -0.6459233164787292, 1.3161276578903198, -1.269314169883728, -0.21067646145820618, -1.5184037685394287, 1.2732446193695068, -1.0100151300430298, -1.9365556240081787, 0.01725846529006958, 0.6258883476257324, -1.0938208103179932, -1.2874023914337158, 1.1988186836242676, -1.6520861387252808, -0.10577604919672012, -0.2640755772590637, 0.3518902063369751, -1.3537482023239136, -1.0226200819015503, 0.46303510665893555, 0.21832986176013947, 1.71873939037323, 1.776160478591919, 0.43619829416275024, 0.42055875062942505, -0.01493794471025467, -0.5343988537788391, -0.6848739981651306, 1.4324666261672974, -0.8108224272727966, 1.2039059400558472, 0.8008014559745789, -0.3597984313964844, 1.3993767499923706, 0.6873098611831665, -0.06937476992607117, 0.08569306135177612, 1.3444688320159912, -0.60240238904953, -1.2297484874725342, -0.2746036648750305, -0.6252173185348511, -0.4358222186565399, -1.0204929113388062], [0.056327980011701584, -0.9945452809333801, 0.4890660345554352, -0.6481270790100098, -0.5350959300994873, 0.1298343688249588, 1.0385925769805908, 2.500864267349243, -0.6172560453414917, -0.6863730549812317, 0.18366287648677826, -0.5699665546417236, 1.1510322093963623, 0.9186589121818542, -0.3875294625759125, -2.560068130493164, -0.1227302998304367, -0.5843760967254639, -0.22263400256633759, 0.9867513179779053, 2.178525686264038, 2.2782437801361084, 1.278795599937439, 0.25577524304389954, -0.9171217083930969, -0.1928885281085968, 0.9645764827728271, -0.48957717418670654, -1.114335060119629, -0.17009767889976501, 1.1023192405700684, 1.7378076314926147, -0.6729176044464111, -1.2113391160964966, 0.8567708134651184, -1.5219683647155762, -0.9967507123947144, -1.3495609760284424, 0.14123529195785522, 1.4261281490325928, -0.25844332575798035, 0.10694031417369843, 0.18188594281673431, -0.12806996703147888, -0.132744699716568, 1.1020550727844238, -0.5101895332336426, 2.795278787612915, 1.0753264427185059, 1.3425328731536865, 1.045641303062439, 0.8383933901786804, 0.6241698861122131, 2.1643593311309814, -0.5782721042633057, 0.006377306301146746, 0.5403680205345154, 0.20001943409442902, 0.945406973361969, -0.33985283970832825, -0.029276778921484947, 0.5078146457672119, -0.3841257393360138, 1.540567398071289], [2.989372730255127, -0.15046250820159912, -1.5910425186157227, 1.9183834791183472, 1.1713894605636597, -0.4071144461631775, 1.768019676208496, -0.002323815366253257, 0.3335106372833252, -1.5455089807510376, -0.47706589102745056, -0.17999868094921112, 0.5448391437530518, -1.5700690746307373, 0.7892547249794006, 0.36191797256469727, 0.026858298107981682, 0.9029420018196106, 0.6166741847991943, 0.018361149355769157, 1.1535700559616089, -0.3273254334926605, -0.055423106998205185, -0.7544547915458679, -0.7827389240264893, 0.31020718812942505, -0.04483029618859291, 0.7743123769760132, -0.6488092541694641, -0.9718027710914612, -0.1645907461643219, -0.6795870065689087, 0.17290060222148895, 1.1076853275299072, 1.587319254875183, 1.5572240352630615, -0.4534626305103302, 0.059340864419937134, -0.10007323324680328, 0.5802642107009888, 0.7176595330238342, 1.2416651248931885, 0.18893581628799438, 1.202293872833252, 1.8615261316299438, 0.4795086085796356, -0.43147289752960205, -1.0260577201843262, 0.7859065532684326, 1.0137346982955933, -1.0568339824676514, 1.4377213716506958, 0.33418896794319153, -0.010817127302289009, -0.33901846408843994, -0.3592853844165802, 0.33517637848854065, -0.7113739848136902, 1.0563676357269287, -1.1322335004806519, 0.6895200610160828, -0.21002653241157532, 1.614868402481079, -1.5034151077270508], [0.012808634899556637, 0.2952656149864197, 0.025404397398233414, 0.7012383341789246, 0.15092076361179352, -0.07855267822742462, -1.5178930759429932, -1.0301661491394043, 0.8371953368186951, -2.0684943199157715, -0.3926996886730194, -0.1402857005596161, -1.451403260231018, -0.5600871443748474, 1.3042210340499878, -0.08775492012500763, 0.7613030672073364, 0.22219112515449524, -2.334439754486084, 0.8734399080276489, 0.8169823884963989, 0.29433995485305786, -1.2480411529541016, 1.576411247253418, -0.7466554641723633, 0.9114874005317688, -0.17056472599506378, -0.2155953198671341, -1.1839344501495361, -0.4489569067955017, 0.6697922945022583, -0.1632636934518814, -0.07773712277412415, 1.9780226945877075, 0.05599908530712128, 1.4188836812973022, -0.8180044293403625, 0.8671760559082031, 1.3585116863250732, 1.0510541200637817, -0.6520838737487793, -0.6642899513244629, -0.09830386191606522, -0.6662768721580505, 0.6969190239906311, 0.23924094438552856, -0.9307200908660889, 0.3319963812828064, 0.2606745958328247, -0.9662798643112183, -1.80010187625885, -0.25047677755355835, 0.7561630606651306, -0.2031746208667755, -0.4612845778465271, 0.7081021666526794, -1.0569021701812744, -1.5349457263946533, -0.06299721449613571, -1.022240400314331, 1.6764540672302246, -0.38634398579597473, -1.060897707939148, -0.22507934272289276], [-0.678674578666687, -1.265912413597107, -0.9149047136306763, 1.95401930809021, -1.7821699380874634, 1.2663706541061401, 0.9847468137741089, 0.6499593257904053, -0.1390119343996048, 1.969822645187378, -1.3712669610977173, 1.7112194299697876, 1.0044673681259155, -0.35499393939971924, -0.42763566970825195, 1.672124981880188, -1.2988110780715942, 1.53011953830719, -1.1612679958343506, 0.019235236570239067, 0.5323126912117004, 0.5294609069824219, -1.2195338010787964, 0.974465548992157, -1.051193118095398, 1.315505027770996, -0.31802135705947876, 1.0112555027008057, 0.2763230502605438, 1.039241075515747, -0.3672354221343994, 0.72549968957901, -0.7778446078300476, -0.7954014539718628, 1.1121159791946411, 0.789398729801178, 0.1252521425485611, -0.34631094336509705, -0.37779149413108826, -0.23024122416973114, -0.03297422453761101, 0.40369313955307007, 0.02397342212498188, 1.263227105140686, 0.14805959165096283, -1.0970591306686401, 0.11905364692211151, -0.3705637454986572, 0.8487953543663025, -0.13924504816532135, -1.195421576499939, -0.19473154842853546, 0.40384575724601746, 0.5113272666931152, 0.7534624338150024, 2.0974667072296143, -0.9525272250175476, -0.9060718417167664, 1.38035249710083, 0.8493495583534241, -1.0617711544036865, 0.04675256460905075, -0.42361903190612793, 0.5139054656028748], [0.05955825746059418, -2.169877529144287, -0.2890072166919708, 0.11163453757762909, 0.46651291847229004, -1.8254382610321045, -0.2944197356700897, 0.011164980940520763, 0.16428738832473755, 1.928861379623413, 1.9574635028839111, -0.5561201572418213, 0.8696672320365906, -1.2350075244903564, -0.012896664440631866, -1.161077857017517, -0.5500339269638062, 1.5198324918746948, 0.12275276333093643, 1.4215208292007446, -1.0985561609268188, -1.0781759023666382, -0.27288818359375, -0.9211729168891907, 0.8781226873397827, -0.3598968982696533, 0.9344939589500427, 1.3342798948287964, -0.21833406388759613, -1.1665855646133423, -2.4168267250061035, 0.035275012254714966, -0.4448329210281372, 0.6535561680793762, -1.5809428691864014, -0.6496902704238892, -0.325158029794693, -0.4111214578151703, -0.5009046196937561, 0.5009086728096008, 0.4509134888648987, -1.8694325685501099, 1.6853560209274292, -1.8238962888717651, 0.7688161730766296, 2.2190909385681152, 0.2858513593673706, 0.3461740016937256, -0.9632716774940491, 0.4454362690448761, -1.6394054889678955, -0.506628155708313, 1.334822177886963, 0.4772041141986847, -0.20648270845413208, -0.8800943493843079, -0.7931228280067444, -0.2914784848690033, 0.5966529846191406, 0.39683860540390015, -0.21312937140464783, 0.8345082998275757, 0.4328204095363617, 0.6750251650810242], [1.3462814092636108, -0.6375554203987122, -1.6363309621810913, 1.3140642642974854, 0.06917628645896912, -0.5983434319496155, 0.10286769270896912, 1.496696949005127, -0.40581411123275757, -2.5525803565979004, -2.4188191890716553, -0.47506359219551086, 2.4064276218414307, 0.05402989313006401, -0.3140101730823517, 0.3973548710346222, 1.252519130706787, 0.8309773802757263, -0.9803158640861511, 0.2836858928203583, 0.40957698225975037, -1.3019189834594727, -0.3885171413421631, 0.5611715316772461, 0.46139395236968994, -0.8954407572746277, 0.1859167218208313, 1.3051849603652954, -0.40861979126930237, 0.5415176153182983, 1.8556891679763794, 0.7985670566558838, 0.9452335834503174, -0.9697355031967163, 0.6061716675758362, 0.7124689221382141, -0.5292698740959167, -0.6952096223831177, -2.934803009033203, -1.241502285003662, 0.6723058223724365, 0.06635228544473648, -1.375280499458313, -0.8600661158561707, -0.0832497850060463, -0.06958341598510742, 0.6638665795326233, 1.0574642419815063, -1.5836012363433838, 0.5163379907608032, 0.2089921087026596, -0.05088835582137108, 0.814167320728302, 0.5684643983840942, -1.7258511781692505, 1.256956696510315, -1.5070141553878784, 0.017851034179329872, -0.8154435157775879, 0.588978111743927, 0.4870295226573944, 0.9801279902458191, 0.994309663772583, -0.7910652756690979], [-0.7677096724510193, -1.0141589641571045, 1.3639978170394897, 1.311061978340149, -1.2889739274978638, -0.9640992283821106, 0.07466863840818405, -0.22136326134204865, -0.08368317782878876, -1.1975715160369873, -0.9716159701347351, 1.4928122758865356, 0.5985101461410522, -1.908529281616211, -0.5977052450180054, 0.25101861357688904, -0.7850127816200256, 0.05605343356728554, 0.32390210032463074, 0.48486900329589844, 1.6477361917495728, -1.9582953453063965, -0.929065465927124, -0.372048944234848, -0.31048843264579773, 0.37875327467918396, 0.1277286261320114, -0.22736786305904388, -0.7567870616912842, -0.714422881603241, 1.9538242816925049, 0.781909704208374, -0.8724238276481628, 0.8790627121925354, 0.052671778947114944, -1.2715414762496948, 0.33288899064064026, 0.946231484413147, -1.117903470993042, -0.19228573143482208, -1.1320693492889404, -1.5531556606292725, 0.1148691326379776, 0.8694241046905518, 0.40587127208709717, 0.4169626235961914, -0.3986218571662903, 0.7815056443214417, -0.03691848739981651, -0.059943512082099915, 1.2666950225830078, 0.32229259610176086, -0.30182382464408875, 1.7854622602462769, 0.5905916690826416, -1.098954200744629, -0.010860187001526356, 2.514376640319824, 0.7231702208518982, -0.2349095344543457, 1.1764768362045288, 0.3482464849948883, 0.624858021736145, 1.0325125455856323], [-0.04019985347986221, 0.16748706996440887, -0.003898831782862544, 0.8040868043899536, -0.41797012090682983, -1.414329171180725, -0.575764536857605, -0.5241332650184631, -1.075081467628479, -0.11474213749170303, -1.5526083707809448, 0.9206328392028809, -0.34262847900390625, 0.5240480899810791, -0.7772584557533264, 0.7461863160133362, -0.2110150307416916, 1.5704118013381958, 0.4532281160354614, 0.3504992723464966, -1.535576343536377, 1.006818413734436, -0.7782163619995117, -0.9266369342803955, -0.09209004789590836, -0.0034639884252101183, 2.070761203765869, -0.29129740595817566, 1.2801692485809326, -0.7809267640113831, 0.025844689458608627, 0.28993913531303406, 0.671455442905426, -0.5522977709770203, 0.597804844379425, -0.280185341835022, 0.34636542201042175, 0.8853010535240173, 0.8362390398979187, -0.713314950466156, -0.8569788932800293, 0.10425110906362534, 0.4578476846218109, -0.07596719264984131, 1.1272027492523193, 0.4294948875904083, 1.2878389358520508, -0.10229835659265518, -1.0046348571777344, 0.7268390655517578, -0.025226866826415062, -1.229313850402832, -0.0773911401629448, 0.1855715960264206, 0.028042549267411232, -0.6576464772224426, 0.7185206413269043, -0.4600324034690857, -1.7495266199111938, 0.6323097348213196, -1.2947944402694702, 0.9414565563201904, -0.13484935462474823, -0.21078412234783173], [-0.20364299416542053, 0.08064410835504532, 1.4623914957046509, -1.1458252668380737, -0.7869815230369568, -0.531903862953186, -0.6672292351722717, -1.131646752357483, -0.7659088969230652, -0.534862756729126, 0.4037231206893921, 1.5281633138656616, 0.535397469997406, 0.22933070361614227, -0.3113865256309509, 0.6264985203742981, -1.060431957244873, -2.039102792739868, 0.056123048067092896, 1.8766560554504395, -0.2888336777687073, -1.864395260810852, -0.2749517858028412, -0.41966989636421204, 1.389974594116211, -0.46282777190208435, 2.0519144535064697, -0.8186590075492859, -0.5658237338066101, -0.033335186541080475, -0.7098658084869385, 0.7809736132621765, -2.2068092823028564, 0.96331387758255, -1.4373825788497925, -1.0785837173461914, -1.1655794382095337, -0.35528427362442017, 0.030760308727622032, 1.3268585205078125, 0.15469218790531158, 0.5918096303939819, -1.3274682760238647, -0.21670064330101013, -0.6740291118621826, -0.4334195852279663, -0.13886719942092896, -1.1621341705322266, 0.20265507698059082, -0.19809839129447937, -0.13752040266990662, -1.6805874109268188, -0.2716625928878784, -0.8825145363807678, 0.022015785798430443, -1.7392890453338623, 1.2161554098129272, -0.2980192303657532, 0.3778746426105499, -1.2534332275390625, -2.334568500518799, -0.2163049727678299, 1.9584068059921265, -0.061568692326545715], [0.39612314105033875, 0.0038208579644560814, -1.2495778799057007, 0.581743597984314, -0.0977788195014, -0.736632764339447, -0.7917339205741882, 0.20206178724765778, 0.8342599868774414, -0.9772382974624634, -1.070127010345459, -0.8883899450302124, 0.07522015273571014, -0.21954724192619324, -0.058598924428224564, 0.25276514887809753, -0.02199937030673027, 0.9522908926010132, -0.1110319271683693, 0.23036503791809082, 0.22817878425121307, 0.865411102771759, 2.323829174041748, -0.5149320960044861, 1.1245160102844238, 1.5887656211853027, 1.1768295764923096, 1.4885709285736084, 0.17975586652755737, -0.04949465021491051, 0.6423576474189758, -1.401437759399414, 1.0872806310653687, -0.6761936545372009, 1.2478697299957275, -0.5920116305351257, -0.4652816355228424, 0.7687802314758301, 2.223712682723999, 0.7619432806968689, -0.34364503622055054, -0.16282056272029877, 0.10459387302398682, 0.2840263247489929, 0.38109245896339417, -0.07582629472017288, 0.5608565807342529, 0.41555067896842957, 0.06680545210838318, 0.16860993206501007, 2.2599637508392334, 0.26313355565071106, 0.5216785073280334, 0.23794859647750854, 0.512919545173645, 0.32208913564682007, 1.5455626249313354, -0.3433111608028412, 0.8618493676185608, -2.247265577316284, 0.13303372263908386, -0.5046361684799194, 0.6323561668395996, 0.41562536358833313], [-0.2449994683265686, -0.8431263566017151, 0.7847260236740112, 1.9862852096557617, 1.739365816116333, 0.39662307500839233, -2.1942315101623535, -0.8648297190666199, 0.6325476169586182, -0.024399515241384506, 0.5177202224731445, -1.3091912269592285, 0.7864394783973694, -0.03594467043876648, 1.6525053977966309, 1.2779818773269653, -0.027049655094742775, -0.7043832540512085, 0.8048205971717834, 0.07534677535295486, 0.7415387630462646, -0.6047849655151367, 0.5199167728424072, -1.271657109260559, -0.42115551233291626, -1.665278673171997, -0.6957917213439941, -1.718182921409607, 0.0762064978480339, -1.1594809293746948, 0.7784348726272583, 0.5153111815452576, 0.3716360628604889, 0.7360567450523376, -1.582901120185852, -1.0482078790664673, -0.7459847927093506, -0.14738333225250244, -0.569470226764679, -0.09373175352811813, 0.5014724731445312, -0.5407600998878479, -0.486726313829422, -0.18175111711025238, -0.16792473196983337, 0.8457203507423401, -0.52748042345047, -1.0764901638031006, -1.4246309995651245, 0.012508336454629898, -0.18783444166183472, 1.0498756170272827, -0.8276604413986206, -1.9800463914871216, 0.06010252982378006, -1.7044968605041504, -1.0808008909225464, 0.22571590542793274, -0.43080201745033264, -1.15830659866333, 2.8535995483398438, -0.1077304258942604, -0.9624432921409607, 0.5283650755882263], [-1.37337064743042, -1.0549038648605347, -1.0289336442947388, -0.5015985369682312, -2.8188271522521973, -1.1958140134811401, -0.9644773006439209, 1.8258458375930786, 0.381607323884964, 0.7204504609107971, 0.6166554689407349, 0.8493218421936035, 1.5068368911743164, -1.6374812126159668, -1.4282808303833008, 2.819807291030884, -1.6611965894699097, -0.049871861934661865, -0.6269891262054443, -1.338078260421753, -0.2481020987033844, 1.0175399780273438, -0.23729445040225983, -1.1939338445663452, -0.13494354486465454, -1.4511346817016602, -0.25847145915031433, -0.3682548999786377, -0.1398317813873291, -0.14390937983989716, -0.12425225973129272, -0.3631344735622406, 0.9561561346054077, -0.2866864800453186, -1.7211675643920898, 0.27070826292037964, 0.17774249613285065, 0.8498797416687012, 1.0462141036987305, 0.5772622227668762, -0.6404194235801697, -0.47964006662368774, -1.2397726774215698, -1.549405813217163, -0.14584359526634216, -0.4323680102825165, -0.5665109157562256, 1.4857330322265625, 0.2968672513961792, -0.10382235050201416, -0.852328360080719, 0.4037474989891052, 1.567379355430603, -0.5526506900787354, -0.5205051302909851, 1.856450080871582, 0.7795491814613342, 2.148568630218506, 0.3214484453201294, -0.8628865480422974, -1.5428961515426636, -1.913649320602417, 1.0707358121871948, 0.6072746515274048], [0.8041961193084717, 0.01602974534034729, -0.04547670856118202, -1.2402517795562744, 0.6513856649398804, -0.1738620400428772, -1.1541035175323486, -0.29514381289482117, -0.7743799686431885, -0.16969804465770721, 1.7813838720321655, 0.3667410910129547, -0.2956736981868744, -0.23314329981803894, -0.06923262774944305, 0.12156815826892853, -1.312323808670044, -0.7648371458053589, -2.5570106506347656, 0.15600474178791046, 0.2786110043525696, -0.06706250458955765, 0.2728196680545807, -0.48987454175949097, 0.5377050638198853, 0.043893761932849884, 0.8438206315040588, 1.291763424873352, 1.059842586517334, -1.79148268699646, 0.3883208930492401, -0.5125054121017456, 0.35127976536750793, -0.40797919034957886, 0.15655966103076935, -0.9084113836288452, -0.29541826248168945, -1.170696496963501, 0.6000049114227295, -2.0897154808044434, 1.2033603191375732, 0.7017434239387512, 0.3287792205810547, 0.6381914019584656, -1.2171108722686768, -2.1349854469299316, 0.18846376240253448, 1.0685319900512695, 0.1690119206905365, -0.7226034998893738, 0.7580872178077698, 1.7283778190612793, -0.0015367921441793442, 0.3040965795516968, 0.18535029888153076, -0.5911269783973694, 0.67665034532547, -0.8183436989784241, 0.447585791349411, 0.08649570494890213, 0.40909186005592346, 1.1873403787612915, 1.351320743560791, -1.270959496498108], [-0.3406040072441101, 1.3757894039154053, -0.17796571552753448, -2.171753168106079, 2.379436492919922, -1.5109835863113403, 0.24567537009716034, -1.4433636665344238, 1.0601366758346558, -0.3226597011089325, 0.054038453847169876, 0.6056264638900757, 0.013905939646065235, -0.41248998045921326, -1.6397027969360352, -1.7760618925094604, 0.5823124051094055, 0.692225992679596, 0.3778045177459717, 0.4293363690376282, 0.8929035663604736, 0.5564656257629395, -0.35238155722618103, -1.244284987449646, -0.6487127542495728, -2.1348252296447754, -1.2560293674468994, -1.9665042161941528, -1.2794697284698486, -0.6778046488761902, -1.361860752105713, 0.7058097124099731, 1.0312232971191406, 0.5372477769851685, 0.08797566592693329, 0.5512233972549438, -0.04198628291487694, 1.6564640998840332, -0.7854561805725098, 0.8760734796524048, 0.45188602805137634, -0.2172577679157257, -1.20844304561615, -1.1683002710342407, 0.17906422913074493, 0.8414930105209351, -0.2946862578392029, -1.2013503313064575, -1.1981319189071655, 1.004916787147522, 0.5273498296737671, -0.07637936621904373, 0.836344301700592, 0.7192651629447937, 0.3586921989917755, -1.2238986492156982, -0.6257752180099487, -1.2571358680725098, 0.7515258193016052, 1.1948471069335938, 0.8615681529045105, 1.1214243173599243, -0.752179741859436, -0.6182394027709961], [-0.5758723616600037, 1.573672890663147, -0.7356976270675659, 1.1085193157196045, -0.4838811755180359, -1.1059678792953491, -0.024703864008188248, 1.4779574871063232, 0.5547805428504944, 0.8972655534744263, 0.794292688369751, -0.8633714318275452, -1.1171613931655884, 0.792133629322052, -0.41469982266426086, 0.7871093153953552, -0.6498785018920898, 0.3490583002567291, -1.3295259475708008, 0.02240612730383873, -1.1436189413070679, 1.7333329916000366, 0.7310402393341064, 0.9399327635765076, -0.2290002405643463, 0.8751969933509827, -1.659336805343628, 0.8975584506988525, 0.619705080986023, -1.1581333875656128, 0.405892938375473, -0.14230787754058838, -0.4395756423473358, -0.6380094289779663, 0.6688335537910461, 1.5386534929275513, -0.8556630611419678, 2.032607078552246, -1.2185221910476685, -0.9008930325508118, 1.182247519493103, -0.8539232015609741, 1.0770496129989624, 1.0661271810531616, -0.8864074945449829, -0.07346796989440918, -1.3577377796173096, -0.07441581785678864, -0.03604557737708092, -2.216153144836426, 0.8585621118545532, 0.22585317492485046, -1.3386266231536865, -0.37927213311195374, 0.7221975326538086, 2.136932373046875, -0.21076591312885284, -1.4196538925170898, -1.6044597625732422, 0.302865207195282, -0.18653017282485962, -0.22137689590454102, -1.1731551885604858, -1.391802191734314], [-1.0285295248031616, -0.8354521989822388, -1.3160878419876099, 1.850644826889038, -1.1881747245788574, 0.2916753888130188, 0.43690189719200134, -0.8438240885734558, -0.527073860168457, -1.2436332702636719, -1.0963257551193237, 1.6121593713760376, -0.5799098610877991, -1.0422524213790894, 0.10569348186254501, 1.5785107612609863, 0.28671976923942566, 1.465651512145996, 0.4716903269290924, 0.18293659389019012, 1.3252791166305542, 0.8872244954109192, 1.5678492784500122, -0.46355289220809937, 1.1910938024520874, 0.5730025768280029, -0.5701046586036682, -0.16750836372375488, 1.4952884912490845, 0.3987336754798889, -0.9805916547775269, -2.0112006664276123, 1.8486064672470093, 0.7092311978340149, -0.35230615735054016, -2.4160261154174805, 0.2775431275367737, 1.0770641565322876, 1.2158149480819702, 0.26126784086227417, 0.44190555810928345, -1.475785732269287, 0.6353549361228943, 0.9637640118598938, 0.156664177775383, -1.7820900678634644, 0.6927565932273865, 0.5039171576499939, 0.249592587351799, 0.3828648626804352, 1.521342396736145, -0.027145924046635628, -1.4299358129501343, -1.487156867980957, -0.8120470643043518, 1.2009161710739136, -0.8281211853027344, 1.1573728322982788, 1.7123174667358398, -1.5943331718444824, 0.5408467054367065, 0.0288659930229187, 1.1560008525848389, -0.5924373865127563], [-0.06946457922458649, 0.8033971190452576, -0.1758512705564499, 1.3161139488220215, -0.5002656579017639, 1.34476900100708, 2.533147096633911, -2.4897313117980957, -1.0212254524230957, -0.5533846616744995, -0.6669721007347107, 0.3207346498966217, 0.6853217482566833, 0.3238634169101715, -0.31663432717323303, -2.41687273979187, -0.9210696816444397, -0.12066657096147537, 0.9844682812690735, -0.3837640583515167, 0.7902863025665283, 0.4158311188220978, -1.0606063604354858, 0.06383998692035675, 0.8954272866249084, -0.8999837636947632, 1.7580170631408691, 0.9800524115562439, -1.442474126815796, -0.03287447616457939, 0.6508074402809143, -1.7768805027008057, 0.9667576551437378, 1.1257729530334473, 0.7376341819763184, 0.41422945261001587, 0.7714940309524536, 0.6780633330345154, -0.08273078501224518, -0.26962387561798096, -0.47215762734413147, -0.6315136551856995, -0.8782595992088318, 1.1243146657943726, -1.1051366329193115, -0.9009509086608887, 0.001027691294439137, 0.29658278822898865, -0.5474743247032166, 0.34144869446754456, -2.20166277885437, -0.21381714940071106, 0.4550285041332245, 0.8560323715209961, 0.4651952087879181, 1.2209908962249756, -0.6066145896911621, -1.3066937923431396, 1.5688323974609375, -1.4813015460968018, -1.0405765771865845, -1.0482851266860962, 1.2979520559310913, -0.174509659409523], [0.09590944647789001, 1.4548296928405762, -1.860547661781311, -0.5748357176780701, -0.4099777638912201, 1.0499927997589111, -1.481924057006836, -0.15044483542442322, 0.8540208339691162, -0.5414933562278748, 1.2478255033493042, -3.2363667488098145, -0.19140952825546265, 0.8442954421043396, 1.0397813320159912, -1.341632604598999, 1.1638389825820923, -0.14983731508255005, 0.9817607998847961, 3.0164334774017334, -0.8622646331787109, -0.25333213806152344, -0.23644854128360748, 1.0445287227630615, -1.4151240587234497, 1.453367829322815, -0.768173098564148, -0.33515337109565735, -0.5835669636726379, 2.330629825592041, -0.9862281680107117, -0.1702599972486496, -0.46457764506340027, 1.122543215751648, -0.9634010791778564, -0.6457008123397827, 0.8162112832069397, -0.962028443813324, -0.42966198921203613, -1.1603461503982544, 0.6348345279693604, -1.3374483585357666, -0.3123682737350464, 1.2349882125854492, 0.2598842680454254, -0.4200887680053711, 1.1906943321228027, 0.9077001214027405, -1.284080147743225, 0.09039292484521866, -1.9119794368743896, 0.5442793965339661, -1.4885584115982056, -0.741077721118927, -0.2352617084980011, -0.42092955112457275, 0.6737997531890869, -1.6456074714660645, -0.11504675447940826, 0.24738281965255737, -0.9827661514282227, -0.273843377828598, 0.09432829171419144, 0.7644741535186768], [-0.5304303765296936, -0.9943619966506958, -0.22989347577095032, 0.7644367814064026, 1.5312250852584839, -0.9946283102035522, 0.8255079984664917, -0.6331400275230408, 0.22485743463039398, 1.1885663270950317, -0.6388010382652283, 0.10465449839830399, -1.5163698196411133, -0.29921755194664, -0.7199419736862183, -0.37678825855255127, -0.42436397075653076, 0.6615186333656311, 1.369802474975586, 0.9346790313720703, -0.5925204753875732, 1.4486645460128784, 0.03865756466984749, -0.19406865537166595, 0.7122874855995178, 1.4877586364746094, 2.2519314289093018, -1.0482600927352905, 0.660636305809021, -0.3412390649318695, -0.24467575550079346, -1.180469274520874, -0.5316807627677917, -1.5672330856323242, -1.531062364578247, 1.0710206031799316, -0.9565442204475403, -1.8634012937545776, 0.07869520038366318, 0.3032916784286499, -0.16792014241218567, 0.5182371139526367, -0.6948572993278503, 0.41620007157325745, 1.5725445747375488, -1.6086195707321167, -1.9438631534576416, -0.031873803585767746, 0.9587481617927551, -0.24161991477012634, 0.4077358841896057, -1.4450817108154297, 1.9417804479599, -0.7749941945075989, 0.9797062277793884, 0.011535700410604477, 0.3392523229122162, 0.5198909640312195, -0.2805754840373993, 0.4487060308456421, 1.2247170209884644, 0.3608371317386627, 1.340929627418518, -0.11204682290554047], [0.34325316548347473, -0.3089369833469391, -1.2876120805740356, 1.4934563636779785, -0.5381762385368347, 1.213469386100769, 1.3228603601455688, 1.55630624294281, -1.4957975149154663, 0.9792957901954651, 1.0276522636413574, -0.6199079751968384, -1.0175633430480957, 1.3932963609695435, 0.2782309949398041, 0.6957680583000183, 0.9177765846252441, 0.7713562250137329, 0.7520507574081421, -1.5782787799835205, -1.8440320491790771, 1.0579298734664917, 1.3188703060150146, 0.619785487651825, -0.26012367010116577, -0.6719337105751038, -0.8577411770820618, -1.6055841445922852, 0.7953051924705505, 0.23191575706005096, -1.800464391708374, 0.5532167553901672, 0.2440669983625412, 0.33235445618629456, -0.7668576836585999, 1.7765388488769531, 1.2606369256973267, 1.576095461845398, 0.36766037344932556, 0.04669884592294693, -0.6525893807411194, 1.2633248567581177, 0.5770240426063538, -0.6290819644927979, -1.012384057044983, 0.9956572651863098, 0.37548524141311646, -0.6983759999275208, -1.0417158603668213, -1.3313096761703491, -1.4874162673950195, -0.10164833068847656, 0.6638290882110596, 0.5982415676116943, -0.1638535112142563, 1.0731207132339478, -0.496722936630249, -0.20860444009304047, 1.1560275554656982, 1.0927917957305908, -1.7195327281951904, 1.7107064723968506, -0.488209068775177, 0.7781392335891724], [0.5930943489074707, -0.9524533152580261, -0.6836851239204407, -0.08460429310798645, 0.05977656692266464, 0.12584421038627625, -0.8090633153915405, -0.3761105239391327, 0.0033882472198456526, -0.23621588945388794, 1.5625742673873901, -0.5285044312477112, -0.15692713856697083, -0.5274993181228638, -0.9750462770462036, 1.042000412940979, -0.4026729464530945, 0.20800048112869263, -0.13625909388065338, 0.64985191822052, 0.6431908011436462, 0.16542664170265198, -0.025945862755179405, -0.1799367368221283, 0.29157090187072754, -1.4155256748199463, 2.1523044109344482, 0.7323048710823059, 2.5228168964385986, 0.1856396496295929, -2.1252405643463135, -0.5890398025512695, 0.3526068925857544, -1.083325982093811, -1.051647663116455, -1.7812877893447876, -0.5807372331619263, 0.6504279375076294, 0.21019263565540314, 1.014151930809021, 0.21622879803180695, 0.6046985387802124, 1.4314278364181519, 0.7136473655700684, 0.8222154378890991, 0.8359678983688354, -0.6284491419792175, -0.45639634132385254, -0.5341216325759888, -0.38794267177581787, -0.4835100769996643, -1.1142548322677612, 2.1132943630218506, 1.076645851135254, 0.12312888354063034, -1.2072315216064453, -0.5999422669410706, 1.4297970533370972, -2.051769733428955, 0.9112796187400818, -1.164340615272522, -0.8552860021591187, 1.7702571153640747, -0.6838245391845703], [2.038597345352173, -0.513008177280426, 1.083829402923584, -0.8365548849105835, -1.1382560729980469, 1.0909299850463867, 0.1301976442337036, 0.7613462805747986, 1.6119375228881836, 0.6804266571998596, -2.622447967529297, 0.9536915421485901, -0.3132854402065277, 0.34209996461868286, -0.3808481693267822, 0.24474048614501953, 0.6973292231559753, 1.0171860456466675, 0.7774744629859924, -0.6820849180221558, 0.8628824353218079, -0.3837318420410156, 0.2725997269153595, -1.3167550563812256, 0.5390976071357727, -2.2274580001831055, 2.0785558223724365, 0.5078206062316895, -1.3757089376449585, -0.3599015772342682, -0.28758856654167175, -0.009483308531343937, 0.7112782597541809, 1.0297445058822632, -2.1604785919189453, 0.334150493144989, -0.10210683196783066, -0.07135261595249176, 0.07429488003253937, 0.5770503282546997, 0.3972513973712921, -0.11598663777112961, -0.9472960233688354, -0.8592967987060547, 1.54142427444458, 0.15573832392692566, -0.3975314795970917, -1.4287846088409424, -0.5247343182563782, 1.262223482131958, 2.3292269706726074, -1.866208791732788, -0.4794008731842041, 1.0843740701675415, 0.16552554070949554, -1.2353612184524536, 0.09400797635316849, -0.708063006401062, -0.5599004626274109, -1.7594382762908936, 1.5047310590744019, -0.7780494689941406, 0.6640602350234985, 1.4422614574432373], [-0.4807775020599365, 1.2596473693847656, 0.4934123456478119, -0.2306295484304428, -0.7851381897926331, 0.34451988339424133, -1.4264012575149536, 0.6335448622703552, 0.27138903737068176, 0.5225926041603088, -0.3075028955936432, -0.8766859769821167, 1.6974077224731445, -1.0901639461517334, 1.1271207332611084, -1.6122777462005615, -0.008660899475216866, 0.043489448726177216, 0.07805407792329788, -0.987095296382904, -1.0498764514923096, 0.19988848268985748, 2.185842752456665, 0.7496236562728882, 0.3625316917896271, 1.3560627698898315, 0.6969695687294006, -1.8352231979370117, -0.5416722893714905, -0.14282608032226562, -0.6715470552444458, -0.574038028717041, -2.197157382965088, 2.5470383167266846, -0.4331539571285248, -0.37918972969055176, -1.1447392702102661, -0.6861892938613892, -0.306241899728775, 0.6987341046333313, 0.567025899887085, -0.2852492034435272, -0.9112650156021118, 0.39190804958343506, 0.4078161120414734, -1.3738620281219482, -2.852506399154663, -2.4995269775390625, -0.4493127465248108, 0.0577688030898571, 1.050260305404663, -0.7067790031433105, 1.0102698802947998, -1.4351799488067627, 0.059267573058605194, 0.5120658874511719, -0.9282996654510498, 0.4863830804824829, -0.2530946135520935, -0.7440136075019836, -0.39881420135498047, -0.4237723648548126, 0.16293840110301971, -0.15948008000850677], [0.6646795868873596, -0.5335248112678528, -0.07670296728610992, 1.2204749584197998, -1.0019432306289673, -0.19308419525623322, -1.0714000463485718, 0.3361256420612335, -1.764363408088684, -0.7605715990066528, 0.21596549451351166, 0.8007829785346985, 0.0674934908747673, 1.0186957120895386, -0.07007230818271637, -0.8643379211425781, 0.0918436348438263, 1.1634297370910645, -1.4565540552139282, -0.4281599521636963, -0.9319751858711243, 0.06611615419387817, -0.41987502574920654, -2.1088900566101074, 1.9676785469055176, 0.9868497252464294, -0.19657324254512787, 0.8051362037658691, -0.3810129463672638, 0.7835639715194702, 0.2739635705947876, -1.3955717086791992, -0.25612300634384155, -0.6796605587005615, 0.5863757729530334, -0.5503891706466675, 1.9649271965026855, 0.21154996752738953, -0.4666039049625397, 0.5329576134681702, 0.6352071166038513, 0.9106560349464417, 0.43059012293815613, 0.322074830532074, 0.24061422049999237, 0.9894915223121643, -0.7735021114349365, 1.864059567451477, 0.345199316740036, -0.6664724946022034, 2.4060962200164795, -0.6490710973739624, 0.18499907851219177, -0.5982146859169006, 0.25884515047073364, 0.3650967478752136, 1.5381207466125488, -1.0433845520019531, 0.20470237731933594, -0.17203722894191742, -0.4891527593135834, 0.7960169315338135, 0.846790611743927, -0.41784897446632385], [-0.9877332448959351, -0.8599369525909424, 2.139441967010498, -1.5288927555084229, 0.03651224076747894, 0.651698887348175, -0.34923598170280457, 0.5030088424682617, 0.07136670500040054, -0.03669341281056404, -0.40749722719192505, -2.3069117069244385, -1.3573607206344604, -0.36150118708610535, -1.110569715499878, -0.909229576587677, 0.9108231663703918, 0.6546126008033752, -0.5427227020263672, 0.12671124935150146, 1.030964970588684, 0.44092491269111633, -0.9194779396057129, -0.34976211190223694, -0.7350504398345947, 0.2771092355251312, -0.338168740272522, 1.1070396900177002, -1.25531804561615, 0.7976758480072021, -0.29121312499046326, -0.6427385210990906, -0.314227819442749, 1.7128040790557861, -1.3915098905563354, -0.1287969946861267, -0.027640029788017273, 0.3191418945789337, -0.43107807636260986, -0.7202016115188599, 0.9625413417816162, 1.454285979270935, -0.5829409956932068, -1.495208978652954, 0.36127161979675293, 0.13851511478424072, 0.43176552653312683, -1.22461998462677, 0.011170020326972008, 0.7847310900688171, 0.08427614718675613, -1.572468876838684, 1.093757152557373, -0.44257956743240356, -1.1287691593170166, -0.6616371273994446, -0.7381253242492676, -0.014097260311245918, -1.0872639417648315, 0.2720828652381897, 1.3655853271484375, -2.534111738204956, 1.1226972341537476, 1.9647963047027588], [-0.8952492475509644, 0.6022638082504272, -1.02263343334198, -0.6709098815917969, 0.10358697175979614, -1.4483981132507324, 0.6831030249595642, -3.2951855659484863, -0.7378314733505249, 1.4107922315597534, 0.7010553479194641, -0.9663171768188477, -1.3231600522994995, 1.0853428840637207, -0.0853176862001419, 0.9212693572044373, 0.5449782013893127, 0.42368432879447937, 0.7596015930175781, 1.429685115814209, -0.08606243878602982, 0.36892983317375183, 1.090444803237915, 1.6006224155426025, 0.06818530708551407, 0.8647503852844238, 0.7685021162033081, -0.45883095264434814, 0.08055035769939423, -0.20243212580680847, -0.5227370858192444, -0.5414234399795532, -4.210337162017822, -1.3617185354232788, -0.1456158459186554, -0.6100662350654602, 0.5830159783363342, -0.8515670895576477, 0.5781324505805969, 0.755845308303833, 0.007795494981110096, -1.1638295650482178, 1.926959753036499, 0.5140249133110046, -0.03152084723114967, -0.8670536279678345, -0.9128530025482178, 0.5396580100059509, 0.3323456048965454, 1.3237978219985962, -0.24998003244400024, 0.7690016031265259, 1.2559940814971924, 1.54708731174469, -1.2258164882659912, 0.0628313422203064, 1.2251403331756592, 0.5447899103164673, 0.3810419738292694, -0.4176480174064636, -0.36769166588783264, -2.8176674842834473, -0.44905465841293335, -1.6379586458206177], [-1.1477676630020142, -0.735626757144928, 0.023791085928678513, -0.17625294625759125, -0.9261485934257507, -0.19056713581085205, 0.04524067044258118, -0.36959654092788696, -0.5635600090026855, 0.1641126275062561, -0.42595258355140686, 1.077000617980957, 1.4964262247085571, 0.06930578500032425, -0.3047935962677002, 0.05423242971301079, 1.1364027261734009, -1.0514521598815918, 0.9607760310173035, 0.7443538308143616, 0.2636817693710327, -0.8371937870979309, -1.2165741920471191, 0.8827341794967651, 1.012940526008606, -1.9604960680007935, -0.8692688345909119, 1.5797709226608276, 0.24928238987922668, 1.2315757274627686, 0.5412397384643555, -0.4499613046646118, -2.1491281986236572, -0.23351217806339264, 0.5952476859092712, 1.8664695024490356, -0.1576230227947235, 1.075376272201538, 1.2130546569824219, -0.9559311866760254, 0.44827255606651306, -0.2778880000114441, 0.00023557186068501323, -0.28181320428848267, -0.25046053528785706, -1.9253392219543457, -0.22195826470851898, -1.615483045578003, 2.24650239944458, 1.2938792705535889, 0.8904834389686584, -0.9534354209899902, -0.5387651920318604, 0.4553494453430176, 0.37155258655548096, 2.0829992294311523, -0.9227704405784607, 0.6738850474357605, 0.32687363028526306, 0.9180355072021484, -1.2457714080810547, -0.6205249428749084, 0.3492324948310852, 0.44928306341171265], [-0.34788766503334045, -0.0036926765460520983, -1.3945364952087402, 0.4657544493675232, -0.2035026252269745, 2.1609320640563965, -0.5982450246810913, -0.7158558368682861, 1.5388057231903076, -0.5787091851234436, -0.16995090246200562, -0.7768322229385376, -1.901815414428711, -0.2997817099094391, 0.8129141926765442, 0.36964502930641174, 1.65627121925354, 2.367982864379883, -0.8287834525108337, 0.4108988046646118, -1.98308265209198, 0.27173948287963867, 0.6537903547286987, 0.07571630924940109, -1.2690106630325317, -1.2066370248794556, -0.24047793447971344, -1.6211538314819336, -0.9073886871337891, -1.3166288137435913, 0.11789096891880035, -1.1541855335235596, -1.061240553855896, 0.9480367302894592, 0.9733097553253174, -0.17973747849464417, -0.4045853614807129, 0.5719166398048401, 1.4832316637039185, -0.5147985219955444, 0.5661775469779968, -1.2252260446548462, 1.1663697957992554, -0.9986069798469543, 0.5017639398574829, 0.6536538004875183, -0.017172763124108315, -0.19400249421596527, -0.38857266306877136, 2.6056182384490967, -0.6683763861656189, -0.6411496996879578, 0.05262849107384682, -0.6008836030960083, 0.11837117373943329, 2.6834521293640137, 0.9374377131462097, -0.647948682308197, -0.9584624171257019, -0.768807590007782, -1.2456011772155762, 1.4283066987991333, -0.48917433619499207, 0.1284603327512741], [0.055625610053539276, 0.30767396092414856, -1.5844155550003052, 0.13243158161640167, -1.205456018447876, -0.6658104658126831, 1.3533810377120972, -0.9938128590583801, 0.23577235639095306, 0.8906564116477966, -0.6237746477127075, 0.2475045621395111, 0.26172348856925964, 1.8019059896469116, -0.6729819774627686, 0.3684619069099426, 0.12954500317573547, -1.716627836227417, 0.24114356935024261, 0.7403681874275208, 0.5813563466072083, 0.9088459610939026, 0.9774986505508423, 1.0527228116989136, -1.4513720273971558, -1.8624368906021118, 0.968414843082428, -0.2846768796443939, 2.0510950088500977, -2.1809825897216797, -2.082319974899292, -0.9942143559455872, -0.2560581862926483, 0.9427164196968079, -1.3625209331512451, 0.6493414044380188, 0.3605087995529175, 0.396308034658432, -0.19939279556274414, 1.5138587951660156, 1.6799410581588745, 0.24453939497470856, 1.158630132675171, 0.6419914364814758, 1.4707711935043335, 0.33523911237716675, 0.05294579640030861, 0.19181235134601593, 2.514756917953491, -0.18367668986320496, -0.22247442603111267, -1.565378189086914, 0.45810025930404663, -0.8025738596916199, 0.8589841723442078, -1.1723763942718506, 0.07520539313554764, 2.2944068908691406, 1.2851074934005737, 0.3055661618709564, 0.06113862618803978, 1.510894536972046, 0.9424523115158081, -1.0152219533920288], [0.6797345876693726, 1.155983328819275, 0.19176051020622253, -1.176595687866211, 1.1072646379470825, 2.1080329418182373, -1.936264991760254, 0.4761141240596771, 0.11101992428302765, 1.4222850799560547, 0.3638123869895935, 0.3895697295665741, 0.37189948558807373, 1.817858338356018, -1.7367171049118042, -1.1263794898986816, 0.021859060972929, 0.02967563085258007, 0.4497852027416229, -0.2771523892879486, -2.4962708950042725, 1.970865249633789, 1.3427015542984009, 1.5639914274215698, 0.4570462107658386, 0.7173444628715515, 0.17220793664455414, 0.5405338406562805, -0.3014792501926422, 0.9402258992195129, -0.7303383946418762, -0.4678272306919098, 0.030004575848579407, 0.7797037959098816, 0.17866931855678558, -2.0638175010681152, 2.5716519355773926, 1.0025089979171753, 1.3291095495224, -0.07046158611774445, 1.4955401420593262, -0.02962738648056984, 1.2817047834396362, -1.4028745889663696, -1.1126322746276855, -1.2779626846313477, 0.5012751817703247, -0.5708001255989075, -1.0365427732467651, -1.2197389602661133, -0.5771836638450623, -0.34575462341308594, -0.86620032787323, -0.8658215403556824, 0.04061539098620415, -0.37690261006355286, -0.026842528954148293, -0.6539141535758972, 0.6061879396438599, 0.3799194395542145, -2.20815110206604, 0.12156497687101364, 0.886329174041748, 0.24742557108402252], [-0.6960347294807434, 1.5012699365615845, -0.28054481744766235, 0.2681613266468048, 2.459846258163452, -1.0654953718185425, -1.213341474533081, -1.3571969270706177, -2.1131057739257812, 0.2888403832912445, 0.6992010474205017, 0.5073758363723755, -0.26156148314476013, 0.5603084564208984, -0.504841148853302, 1.8966240882873535, 1.171751856803894, 1.388452172279358, -1.5925014019012451, -0.2914125621318817, -0.11199397593736649, 0.5526313185691833, -0.32382991909980774, -0.994756281375885, -0.12926143407821655, 0.14025817811489105, -2.2714264392852783, 0.538110613822937, -1.4499106407165527, -0.2996140420436859, -0.2390747368335724, -0.6674714684486389, 0.6010879278182983, -0.792453408241272, -1.3178502321243286, 0.4219997525215149, 0.9967003464698792, -0.1352330893278122, -1.1867526769638062, -1.1417789459228516, -0.013690839521586895, 0.4881090223789215, -1.6033961772918701, -0.7171806693077087, -1.5719292163848877, -1.3863763809204102, 0.2891082763671875, -0.12413069605827332, -1.2749251127243042, -0.885708749294281, -0.05597539618611336, 1.5147037506103516, 0.2235034853219986, 2.2534852027893066, 1.9384856224060059, 0.3394652307033539, 0.6212337017059326, -1.9779924154281616, -0.23514653742313385, 0.13322848081588745, -0.015797743573784828, -0.20372775197029114, 1.3694714307785034, 0.24284493923187256], [-0.30530521273612976, 1.3083205223083496, 0.8557831645011902, -1.562605381011963, -1.8068746328353882, -0.5217535495758057, 0.4948227107524872, 0.3555443584918976, 1.2751169204711914, 0.6858811974525452, 1.073733925819397, -1.2867504358291626, 1.0536184310913086, -0.581245481967926, 0.6410362720489502, -0.3804738223552704, -0.09678616374731064, 1.7392255067825317, -0.5952937006950378, -0.596472442150116, -0.8445342183113098, -0.8176592588424683, 1.4231209754943848, -0.23448944091796875, 1.032235860824585, -0.8559824228286743, 0.26592063903808594, 1.5386745929718018, 1.1609920263290405, -0.9201589226722717, -1.1811714172363281, -1.0434455871582031, 1.452906847000122, 0.13327588140964508, -0.514873743057251, -0.08276388049125671, -0.7145900726318359, -1.7608662843704224, 0.2589823603630066, -0.1968454122543335, -1.299987554550171, -1.0528972148895264, -0.13991673290729523, 0.889785647392273, 1.5156095027923584, 0.569420576095581, 1.1646511554718018, -1.3893417119979858, 0.7346340417861938, -0.10473278164863586, 0.25972089171409607, -0.47732314467430115, 0.7948183417320251, 0.07201189547777176, 1.0387988090515137, -1.3076106309890747, -0.36571264266967773, 1.8056378364562988, 2.38421893119812, -0.8570813536643982, 0.5006664991378784, -1.327047348022461, -0.717081606388092, -2.1376051902770996], [-0.25607702136039734, -0.4228219985961914, 0.9021295309066772, -0.5982022881507874, -1.070662260055542, -0.16034770011901855, -1.1400644779205322, -1.0791279077529907, -0.21026839315891266, -0.22573722898960114, 0.8501923680305481, 0.5195484757423401, 0.3915150761604309, -0.5809692740440369, 0.07749754190444946, -0.8550630807876587, 0.17773619294166565, 0.9093510508537292, -1.6151001453399658, -0.7808713912963867, 0.49058809876441956, 0.18306057155132294, -1.2778321504592896, 0.12801387906074524, 0.31398236751556396, -0.24284957349300385, -1.1049230098724365, -0.1408129632472992, -0.2753399610519409, 1.8398281335830688, 0.49610579013824463, 2.375415563583374, 0.7562131881713867, 1.342781662940979, 0.6791201829910278, 0.9064393043518066, -1.9009071588516235, -0.02005993016064167, -0.2547902762889862, 0.7444578409194946, 0.6412339210510254, -1.8376144170761108, 0.10602205991744995, 0.8364296555519104, 0.687829315662384, 0.29131245613098145, -0.8153802156448364, 0.014058949425816536, 2.163336753845215, -0.1763022392988205, -1.0619316101074219, -0.7361270189285278, -1.4382972717285156, 0.08515160530805588, -0.15304993093013763, 0.5198655128479004, -2.2134127616882324, -0.3998194634914398, 0.3219678997993469, 1.4909441471099854, -1.798994541168213, -1.2029378414154053, 0.7856540083885193, 0.9129903316497803], [0.1648414134979248, -1.572009563446045, 0.4675745368003845, -0.2979253828525543, 0.05328645929694176, -1.0362826585769653, 0.2955220341682434, -0.810843288898468, 0.1800002157688141, 0.39907681941986084, -0.17736215889453888, 0.30154478549957275, -1.16820228099823, 0.8580575585365295, 0.21053649485111237, 0.38124144077301025, -0.1432747095823288, 1.8093620538711548, 0.4882209599018097, 0.1986394077539444, -0.6652386784553528, -0.9843813180923462, -1.1419169902801514, -0.44214874505996704, 0.1490560919046402, -0.6255017518997192, 0.8410336971282959, -1.1622605323791504, -0.19687685370445251, 0.10393523424863815, -1.7362996339797974, -1.1630034446716309, 0.45994043350219727, 0.6737489700317383, 0.6158717274665833, 1.0929783582687378, -2.3220407962799072, -1.0746222734451294, 1.6306817531585693, 1.0266119241714478, 0.9620486497879028, 1.3048216104507446, -1.3505218029022217, -1.2536605596542358, -0.8023263216018677, -0.9975059032440186, 2.1826040744781494, -1.34787917137146, -0.3783110976219177, -0.44853001832962036, 0.9246715903282166, -0.3990049362182617, -0.006524698808789253, -1.6054490804672241, 0.023075759410858154, -2.211442470550537, -0.7528120279312134, -0.40435540676116943, -1.5340559482574463, -0.26538410782814026, 0.5059112906455994, 1.2229382991790771, 0.41316452622413635, -0.22758355736732483], [-0.23866823315620422, -0.01823313906788826, -0.15420278906822205, -0.020391616970300674, -0.07897700369358063, 1.3874746561050415, 0.7387063503265381, -0.46903958916664124, -0.14884507656097412, -1.2789692878723145, 0.7551205158233643, 0.26952454447746277, 1.550683856010437, -0.3035067617893219, -0.5727866888046265, 0.9450719356536865, 1.1362849473953247, -0.1576196551322937, 0.19618304073810577, -0.7571457624435425, 0.41921624541282654, -0.022828243672847748, 1.2049204111099243, -0.7181947231292725, 1.1033663749694824, 1.2104130983352661, -0.07387299090623856, -0.8590434789657593, 1.5143423080444336, 1.297066569328308, -0.6865723729133606, -1.780426025390625, 0.007623470388352871, -1.3602877855300903, 0.6348538994789124, 0.9008963108062744, -0.39366772770881653, -0.8497401475906372, -1.3852348327636719, 0.8463027477264404, -1.1399080753326416, -0.598441481590271, 1.1368471384048462, -2.224790334701538, 0.32198065519332886, 0.31426599621772766, 2.712672233581543, -0.920862078666687, -0.42429372668266296, -0.6879006028175354, 0.09131618589162827, 0.6211633086204529, -0.578139066696167, 0.6799203157424927, 0.7600113153457642, -0.02850402519106865, 0.44083571434020996, -0.01497636642307043, -0.3900296092033386, -0.8517314791679382, -0.776964008808136, 1.5808196067810059, -0.9969013333320618, -1.3408886194229126], [1.0170761346817017, 1.6927976608276367, -0.35606157779693604, 0.3031908869743347, -0.9897614121437073, -0.07025553286075592, -0.3918212056159973, 0.943148136138916, -0.8132405877113342, -0.5297442674636841, 0.70233553647995, 1.2071319818496704, 1.1329131126403809, -0.9410862922668457, -0.7902565598487854, -0.20836816728115082, 0.7586466670036316, -0.8098325133323669, 1.0994864702224731, 1.0949815511703491, 0.029678059741854668, -0.9282423257827759, -0.6832776665687561, 0.2817821502685547, 0.8367794752120972, -1.0461100339889526, 0.4804251492023468, 0.4073374271392822, -0.05765809491276741, 0.15730872750282288, -0.4921361804008484, -0.297589510679245, -1.7323352098464966, -0.5639484524726868, 0.3216928541660309, 1.1844120025634766, 0.2386481612920761, 0.9501535296440125, -1.9175641536712646, 0.20770516991615295, 0.6426013112068176, 0.05656740441918373, 0.07484062016010284, 1.301519751548767, -0.7937957644462585, 0.8603405952453613, -0.20062468945980072, -1.4117521047592163, 0.46714866161346436, -0.31307896971702576, -1.284476399421692, 1.307604432106018, 0.8077346682548523, 0.5922282338142395, 2.4563894271850586, -2.2846806049346924, -1.6142044067382812, -0.7400181889533997, -2.3734631538391113, 0.8455454707145691, -0.6275217533111572, 1.0810649394989014, 0.24771320819854736, -0.8316599130630493], [1.2218198776245117, 2.1876792907714844, -0.4265802502632141, -0.4138827323913574, 0.313252329826355, -1.0063388347625732, -0.26639407873153687, 1.2877922058105469, -1.1178267002105713, -0.804039478302002, 0.4110218286514282, 2.1705572605133057, -1.9212226867675781, 1.3138322830200195, 0.45920073986053467, 0.9212741255760193, -0.6794259548187256, 1.2839949131011963, 0.09101570397615433, -1.6873774528503418, -2.5086281299591064, -2.186464309692383, 0.5633378028869629, -0.5456140041351318, -0.9066025018692017, -0.17684105038642883, 0.8740304112434387, 0.5060714483261108, -0.6540650725364685, -0.4699864089488983, -0.7907267212867737, 0.5112453103065491, 0.28059741854667664, 0.012544550932943821, -1.2295771837234497, 0.0007544346735812724, 0.8326753973960876, -0.6286399364471436, 2.9791977405548096, -0.78616863489151, 0.30108028650283813, 0.17212924361228943, 0.29884034395217896, 0.08556680381298065, -0.8388340473175049, -1.669234037399292, -0.7342450022697449, -1.0162365436553955, 1.5149600505828857, -0.22980092465877533, -1.134083867073059, 1.443664312362671, -1.7930907011032104, -0.5697727203369141, 0.59784996509552, -0.09145001322031021, -2.080580949783325, 0.7135385870933533, -0.4052562117576599, -0.02860463596880436, 0.846470296382904, -1.054187297821045, -1.5874384641647339, 1.0630698204040527], [-1.3489434719085693, -0.7714743614196777, -0.5276532769203186, -1.6056489944458008, 1.1629630327224731, 0.35804808139801025, -0.33510515093803406, 2.9605445861816406, 0.9903807044029236, -0.7062491178512573, -2.752164602279663, -0.354058176279068, -0.3384518325328827, -0.6218064427375793, -0.3851586580276489, -0.8557688593864441, 0.675948441028595, 0.5867263674736023, 0.4178723692893982, 0.7342575192451477, -0.7857697010040283, -1.2416226863861084, 0.6393327116966248, -1.242404580116272, -0.2739185690879822, 0.09666620939970016, -0.5969319343566895, 1.2598152160644531, 0.40952107310295105, 1.4480063915252686, -0.5406081080436707, -0.0059696389362216, -1.8586002588272095, 1.2540613412857056, -1.1181749105453491, -0.32836416363716125, 0.48512858152389526, 2.569566249847412, -0.8384709358215332, -0.7679544687271118, -0.19692419469356537, -0.09445402771234512, 1.8267089128494263, 0.16160058975219727, -1.0018975734710693, -0.4458564221858978, 0.12080749124288559, 0.1234685629606247, 0.4051966071128845, -1.2683970928192139, 0.4358724057674408, 0.19596807658672333, 0.3579607307910919, 1.264447569847107, 0.9122557640075684, 1.3576607704162598, -0.05896038934588432, -0.7884624004364014, -0.5174102187156677, -0.14536048471927643, -2.333904504776001, 1.3484684228897095, -0.9985314607620239, -0.9315179586410522], [0.5200099349021912, -1.273581624031067, -0.2664321959018707, 2.18754506111145, -1.8758249282836914, -0.5195264220237732, 1.0771957635879517, -0.6827719211578369, 1.4419292211532593, 0.42278188467025757, -1.4968132972717285, -0.7226419448852539, -0.033183932304382324, 0.4331963360309601, 1.1719290018081665, -0.4395923614501953, -0.6546508073806763, -0.20754840970039368, -1.4804762601852417, -1.3681062459945679, 0.3454519510269165, 0.600391685962677, -0.8999895453453064, 0.22574694454669952, -0.45775339007377625, -1.7771270275115967, -0.7139003872871399, -0.10681866854429245, -0.4190078377723694, 0.3436199724674225, 1.2438358068466187, -1.1983190774917603, -2.1746232509613037, -0.4472866654396057, 0.5085858106613159, 0.6543234586715698, -0.1745835542678833, 1.2888273000717163, -1.2540253400802612, 1.0688477754592896, 1.2289338111877441, 0.5385957956314087, -1.6926511526107788, -2.095409631729126, 1.9257633686065674, 0.6178752183914185, 0.40741831064224243, -0.8527073860168457, -1.0417230129241943, 0.11548035591840744, -1.244705080986023, -0.508580207824707, 0.7195073366165161, 0.9880003333091736, 0.16685643792152405, -2.5698742866516113, -0.7411510348320007, 0.5743523836135864, -1.6657309532165527, -1.1743793487548828, 1.857717752456665, 0.4311531186103821, -1.0268714427947998, 0.7962875962257385], [0.7316040396690369, -2.406200885772705, -1.3564815521240234, 2.7952048778533936, 0.34143760800361633, 0.36207374930381775, -0.20352394878864288, -0.47735297679901123, 0.13138385117053986, 0.5806972980499268, 0.20444171130657196, -0.27899622917175293, -0.7121389508247375, -1.498064398765564, -0.9492012858390808, 0.5957129001617432, -0.6067008972167969, -0.46180593967437744, -0.5121859908103943, -0.9445987939834595, -0.12341854721307755, -0.17147433757781982, 2.0912415981292725, -0.5927212834358215, 0.3469912111759186, 1.088775873184204, -0.5760889649391174, -0.9737581014633179, 0.4205970764160156, 1.0179678201675415, 1.047037124633789, -0.3376144766807556, 0.02375967428088188, 1.299320936203003, 0.8432397842407227, 0.009427119046449661, -0.061387799680233, 1.779721736907959, -1.1344772577285767, -0.35302919149398804, 0.2416638880968094, 0.2089005559682846, -2.5217173099517822, 0.8157292604446411, 0.3414914011955261, 0.43971535563468933, 1.3494399785995483, -0.8622834086418152, 0.20111416280269623, 1.5224145650863647, -1.7314825057983398, 0.9116914868354797, 1.5046910047531128, 1.1345418691635132, 0.5089051723480225, 0.7875661849975586, -0.010303745977580547, 1.8182812929153442, 0.38124677538871765, 0.24117423593997955, -1.685611605644226, 0.14498423039913177, 0.5630974173545837, 0.2779487371444702], [-0.8682569861412048, -0.3751259744167328, -0.11014309525489807, -1.4868388175964355, -0.18942341208457947, -0.4202927052974701, -0.22053882479667664, -0.011060532182455063, -0.3260340690612793, -0.9163296222686768, 0.3610782027244568, -0.479243665933609, 0.6684994101524353, 0.8711740374565125, 1.6285068988800049, -1.2760730981826782, 1.0070395469665527, -0.6855450868606567, 0.9439249634742737, -2.447892904281616, 2.7409040927886963, -0.17103694379329681, 0.07286583632230759, 1.0827964544296265, 0.18682879209518433, 0.17861667275428772, -2.35740327835083, 0.365904837846756, 1.0640078783035278, 0.05639839917421341, -0.29675406217575073, -2.8288071155548096, -0.09095387905836105, -0.15064901113510132, -0.7032788395881653, -0.08887346088886261, 0.7409695982933044, -0.6621766686439514, 0.23417919874191284, -1.163403868675232, 0.59872967004776, 0.3460022807121277, 1.1245073080062866, 0.9455022215843201, 0.3193519413471222, -0.28132542967796326, -0.3852204978466034, -0.2196803092956543, -0.8855270743370056, -0.3955294191837311, 1.1040270328521729, -0.4631556570529938, 0.46049776673316956, -0.2388395369052887, 1.0991648435592651, -0.597855269908905, -2.470841646194458, -0.4037396013736725, 1.2511340379714966, 0.9762247800827026, 1.9145745038986206, -0.011382078751921654, -1.3700177669525146, -0.32666015625], [-0.41021808981895447, 0.46413153409957886, 0.6325570344924927, -1.196722388267517, -1.7790727615356445, -0.8173194527626038, 1.57140052318573, -0.9234163761138916, 2.0033061504364014, -1.2051092386245728, 0.6090231537818909, 0.0779891088604927, 1.5362720489501953, -0.604408860206604, 1.789900541305542, 0.6760761141777039, -1.190035343170166, 1.2187700271606445, -0.02184935472905636, 0.6255209445953369, -1.0487741231918335, -0.3399204909801483, 0.08034562319517136, -0.12860915064811707, -0.5423372387886047, -1.033379316329956, 0.8007397651672363, 0.16384992003440857, 0.1715548038482666, 0.03995472937822342, -0.009813344106078148, 0.06912045925855637, -0.8229759931564331, -0.09039587527513504, 1.1334527730941772, 1.3222346305847168, 1.8141547441482544, 1.896924376487732, 1.7565101385116577, -1.1257224082946777, 0.2407868653535843, -1.4206664562225342, -2.553215980529785, -0.6137744188308716, -0.12436079233884811, -1.509024739265442, -0.5121694803237915, -0.9285509586334229, -1.174307107925415, 0.5055575966835022, 1.0167220830917358, 1.2972973585128784, -0.9919264316558838, 1.1341303586959839, -0.3543657064437866, 0.4459039270877838, -0.590233564376831, 1.0956522226333618, 1.7663404941558838, 0.2239694744348526, -0.402991384267807, -0.19951456785202026, -0.39809316396713257, 1.2421183586120605], [0.40295061469078064, -0.6356831192970276, -0.6013229489326477, 0.1053648591041565, 2.6997573375701904, -0.2182512730360031, -0.11392147094011307, 0.9636986255645752, 1.5529240369796753, -1.0402981042861938, -1.6827503442764282, 1.422324299812317, 2.197732448577881, 0.5864709615707397, -0.3664718270301819, 1.244678020477295, -1.4215210676193237, 0.9235098361968994, -1.6032233238220215, 1.2582745552062988, 0.3781692385673523, 0.7205589413642883, -0.5629561543464661, 1.3472989797592163, 1.6752774715423584, -0.5731545090675354, -1.2836772203445435, -0.39641880989074707, 0.6272113919258118, -1.5166248083114624, -1.1752114295959473, -1.0152205228805542, -1.3753997087478638, 0.04157593101263046, 1.0466614961624146, -0.8911755084991455, -1.4982624053955078, -0.5069822669029236, 0.23624278604984283, 0.889524519443512, 0.4134725034236908, 2.430760145187378, 2.2454445362091064, -1.1359792947769165, 2.2409980297088623, -1.1746044158935547, -0.432438462972641, -0.029444823041558266, -0.49920421838760376, -0.32900962233543396, 1.073693037033081, -0.4911506772041321, 1.0356494188308716, 0.07031846791505814, 0.9356498718261719, -0.3125186562538147, 0.04852324351668358, -0.2506988048553467, -0.13017860054969788, -0.7598820328712463, -0.6976602673530579, -0.0902259424328804, -1.687372088432312, -0.27094948291778564], [-0.17077390849590302, -0.16755908727645874, -0.9557615518569946, -1.02741277217865, 0.15639229118824005, -0.0808979868888855, 0.33714884519577026, -0.5594940781593323, 0.9687743782997131, 0.4882204830646515, -0.20947913825511932, -0.030894318595528603, -0.5037475824356079, -1.063389778137207, 0.6698542833328247, -1.3967124223709106, 0.9435453414916992, -1.6646579504013062, -0.2456430047750473, 0.01348681841045618, 0.5878517627716064, -0.57915198802948, -0.6958094239234924, 0.7734858393669128, 0.7316529154777527, 0.4660719931125641, -1.1152247190475464, -0.9974676370620728, -0.5321010947227478, -0.041210826486349106, 1.2799217700958252, 0.283145546913147, -0.5240055322647095, -1.8185374736785889, -0.4097678065299988, 1.4291034936904907, 0.003092364175245166, 0.21387535333633423, -0.7184249758720398, 1.7098313570022583, -0.4007721245288849, 0.5800038576126099, 0.6683365702629089, 0.4450194239616394, 1.180332064628601, 2.2183749675750732, 1.1837522983551025, -0.8268787860870361, 1.378211498260498, 0.8623505234718323, 0.5301224589347839, 0.9402670860290527, -0.7280219197273254, -0.1499234437942505, 1.4114596843719482, -1.8417447805404663, 1.1758935451507568, -1.0459480285644531, -0.17896951735019684, 1.9622833728790283, -0.5276747345924377, 0.0737726241350174, -1.9999819993972778, 0.9824675917625427], [0.7506524920463562, -0.21346567571163177, 0.9950098991394043, -0.023411130532622337, -0.5430479645729065, -0.032581403851509094, 1.1826874017715454, 0.022106515243649483, 1.4745625257492065, 1.3046971559524536, 1.2682243585586548, 1.5947880744934082, -0.36597225069999695, 0.13569259643554688, 0.6133291721343994, -0.06327936798334122, 0.7557432651519775, 0.45925161242485046, 0.4686059355735779, 0.5305274128913879, 0.6220703721046448, 1.4650497436523438, 0.27267810702323914, 0.6517136096954346, -0.5872812867164612, 0.3992289900779724, -0.7023312449455261, 0.8317700028419495, 0.9513862133026123, -0.5355939269065857, 0.7961580157279968, 0.767029881477356, 0.17787301540374756, 1.2179621458053589, -1.3511465787887573, -0.2992302179336548, -1.1513112783432007, -2.6085944175720215, 1.3673619031906128, 0.5955864191055298, 1.211551547050476, 0.7938745021820068, 0.9101459383964539, -1.2452523708343506, 0.3798322379589081, 0.6906688809394836, -1.3562877178192139, 1.383338212966919, -0.4826900362968445, -2.4788601398468018, 0.05201927199959755, -0.30972328782081604, -1.141157865524292, 2.144360303878784, 0.5929126143455505, 0.5530272722244263, -0.08155933767557144, 0.8979806303977966, 0.0865924060344696, -0.02332976460456848, 2.4531760215759277, -2.163900852203369, 0.29466500878334045, -0.0757298618555069], [0.5741405487060547, 0.8257771134376526, -0.04584313929080963, 0.8120476007461548, -0.5233427286148071, 1.4518324136734009, -0.1409846842288971, -1.7436699867248535, -0.11772380024194717, 0.9236180186271667, 0.7629824876785278, -0.43436023592948914, 0.20828227698802948, 0.2413521111011505, -0.02254374325275421, -1.7210322618484497, -0.6946315765380859, 0.1481916755437851, 0.3908799886703491, 0.4923258125782013, 0.911513090133667, -0.23969963192939758, 0.3291389048099518, -0.5273476243019104, -0.5211091041564941, -0.6737801432609558, 0.9298486709594727, -0.4366593062877655, -1.5658930540084839, -0.08084307610988617, -0.049049992114305496, 0.9703267216682434, -0.2552177608013153, -0.4020729959011078, -0.02508390136063099, -1.2482342720031738, -0.911499559879303, -0.4609540104866028, 1.0207600593566895, 0.17826426029205322, -1.8778975009918213, 1.9577323198318481, 1.1537364721298218, -2.1436409950256348, 0.14942479133605957, 2.228111982345581, -0.906477153301239, -0.03983462229371071, 0.4590834677219391, 0.43737566471099854, -1.7874393463134766, 0.599875271320343, -0.4413871765136719, -0.7894988656044006, 1.7407692670822144, 0.521577000617981, -2.052326202392578, 0.008281716145575047, -0.8751516342163086, 0.16117694973945618, -1.8816466331481934, 0.5912327170372009, 0.9036957621574402, -0.3832900822162628], [-0.29828640818595886, -0.22425074875354767, -1.2279024124145508, 0.4247453510761261, -2.712441921234131, 0.22701293230056763, 1.6444770097732544, -0.24693404138088226, 2.016050338745117, 0.68833327293396, 0.704293429851532, 0.4570806622505188, 1.4505854845046997, -1.174134612083435, 0.6504437327384949, 1.5233222246170044, -0.8950849175453186, -0.7103613018989563, 0.43512600660324097, 1.0751932859420776, -0.2525021731853485, -1.3495724201202393, -1.3474857807159424, 1.2520809173583984, -0.2401559054851532, 0.018348313868045807, 0.016368156298995018, -0.3769710659980774, -1.3829498291015625, 2.108503580093384, -2.0710034370422363, -0.1767347753047943, -0.15524643659591675, -2.2385921478271484, -0.0959533080458641, -0.305322527885437, 0.8762432932853699, -1.1522847414016724, 0.48624730110168457, -0.6819152235984802, 0.15948162972927094, 0.960692286491394, 0.6637033820152283, -0.22967568039894104, -0.4570392966270447, -0.9231146574020386, -1.5356405973434448, -0.9554972648620605, -0.20166073739528656, -0.5592373013496399, 0.048341065645217896, -1.3337981700897217, -0.0993080660700798, -1.1395652294158936, 1.9267642498016357, -1.2946200370788574, 0.1642734706401825, -1.057344913482666, 1.3487746715545654, 0.29936590790748596, 1.1446031332015991, -0.9540907740592957, 1.6310229301452637, 0.12774495780467987], [1.1752090454101562, 0.9049257040023804, -0.6187189221382141, -1.1659728288650513, -0.9575996994972229, 1.4973249435424805, -0.3620322644710541, 0.20210038125514984, -0.03222687542438507, -0.2297986000776291, 1.0179671049118042, 2.0192208290100098, 0.8423930406570435, -0.5567858815193176, -1.4368211030960083, -0.9641340970993042, -0.4615066647529602, 0.7792251706123352, 0.20449620485305786, 0.6638875007629395, -0.2395402491092682, -0.30740541219711304, -2.1285440921783447, 0.32778000831604004, -1.163252353668213, 0.26095595955848694, -0.9895024299621582, -2.2081401348114014, -0.3819023668766022, -0.7411847710609436, 0.1479502022266388, 1.881196141242981, 0.8754075765609741, -1.1337337493896484, 1.0403554439544678, -0.19835618138313293, -0.17484046518802643, 0.6768123507499695, 1.422029733657837, 1.059959888458252, 0.31903550028800964, -1.3046367168426514, 0.1379200965166092, 0.7688219547271729, 1.0493358373641968, -0.8143998980522156, -1.278340458869934, -0.2647330164909363, 1.9790897369384766, -0.33205538988113403, -0.0007755846600048244, 0.19022448360919952, 0.23863032460212708, 0.3342110812664032, 0.037918318063020706, -2.0754971504211426, 0.6373158097267151, 0.5427113771438599, -0.40909889340400696, 0.8306865692138672, -0.6514410376548767, -1.1822539567947388, -0.9421924352645874, -1.062052607536316], [1.4061347246170044, 2.3753421306610107, 0.931908130645752, 0.9222028255462646, -0.6059674620628357, -0.8112068772315979, -0.3365856409072876, -0.274858295917511, 0.1745453029870987, 0.6116122603416443, -0.4667297899723053, 2.261453628540039, -0.3289802074432373, -0.9433040022850037, -1.3478014469146729, 0.18491670489311218, 0.9405294060707092, -1.0994771718978882, -1.2588109970092773, 1.0474501848220825, 0.6875438094139099, 0.9890945553779602, -1.7654335498809814, 0.8493276238441467, 0.38074642419815063, 0.306497186422348, 0.49890315532684326, -0.31031331419944763, 0.4551767110824585, 0.6497789025306702, -1.1797964572906494, 0.34638407826423645, -1.305896520614624, -0.4870237708091736, 0.48653125762939453, 1.6408637762069702, -0.04080729931592941, 0.1532391458749771, 0.16284270584583282, -1.0764819383621216, -0.5559234023094177, 0.20113922655582428, -0.22426432371139526, 1.2170135974884033, 0.48857882618904114, 0.4353342354297638, -0.8892669677734375, 0.40261122584342957, 0.28682050108909607, -0.3302723169326782, -0.2994503974914551, 2.4302096366882324, -0.20691710710525513, -1.218959927558899, -0.6348116397857666, -0.30317166447639465, 1.6329970359802246, 0.7999306917190552, -0.6047968864440918, 0.32279765605926514, 2.122635841369629, -0.9785758852958679, -0.8799876570701599, -0.07103341072797775], [-0.8393784761428833, -0.4786430895328522, 1.1128339767456055, -0.9468779563903809, 0.6419059634208679, 0.6556648015975952, 2.740879535675049, -0.9114945530891418, -0.15856069326400757, 1.7074397802352905, 0.5068118572235107, 0.33963271975517273, 0.9202123880386353, -0.03562821075320244, -0.21828041970729828, -0.5740578174591064, 0.23503632843494415, -0.5677774548530579, -1.7603390216827393, -1.4477756023406982, 1.0436370372772217, -0.19597502052783966, 0.11828358471393585, -0.785776674747467, -0.35818517208099365, 1.11690354347229, 0.23441556096076965, -0.7274550795555115, -0.45034438371658325, 2.425260543823242, 0.371498167514801, -0.011161326430737972, -0.5941998958587646, 1.8265700340270996, 1.608368158340454, 0.16655711829662323, 1.0998330116271973, 2.3645870685577393, 0.6013879179954529, 0.5050662159919739, -0.3932811915874481, -1.484315037727356, 0.45491695404052734, 0.5375540852546692, 0.3014059066772461, -0.8646812438964844, 0.22181321680545807, -2.3102316856384277, 0.05754774808883667, -0.7701044678688049, -0.4891488254070282, 0.002599786501377821, 0.9121025800704956, 1.4225136041641235, -0.39105746150016785, 0.03200991824269295, 0.2792952358722687, 0.06894483417272568, -0.5501096844673157, -0.09897249191999435, 0.8126339316368103, 0.362870454788208, 0.9940781593322754, 0.3572256565093994], [0.4314038157463074, -0.7941805124282837, -0.5534214973449707, 0.2927154302597046, -1.3322175741195679, 0.8079453110694885, 0.9134492874145508, -0.6424758434295654, 0.4937252700328827, -0.9441661834716797, -0.5439170002937317, -1.959953784942627, -0.8689113855361938, -0.38172850012779236, 0.2618604004383087, 0.4373581111431122, -1.011771559715271, 2.2921481132507324, -0.9370354413986206, -1.0298712253570557, -0.309400737285614, 0.8001306653022766, 0.8491714596748352, -0.6911268830299377, -0.07172199338674545, -1.1074135303497314, -1.0125430822372437, -1.3356683254241943, -1.0938739776611328, 0.5553398728370667, -0.34216734766960144, 0.7738084197044373, -0.023224057629704475, 1.2864809036254883, 0.7941543459892273, 2.722752094268799, -0.616948127746582, -0.35012388229370117, 0.4438226819038391, 1.1511404514312744, -0.9117510914802551, -1.5079220533370972, -0.08123038709163666, -0.6632788777351379, 0.5075073838233948, -0.28983616828918457, -0.5166398286819458, 0.5562087893486023, 0.17064975202083588, 1.1698293685913086, -0.6494660377502441, 0.1390673816204071, -0.7533023357391357, 0.38727447390556335, -0.4226427674293518, -0.7935845255851746, 0.5348420143127441, -0.15528523921966553, 1.4877657890319824, 2.2362542152404785, -0.48432281613349915, -1.6384788751602173, 0.8638047575950623, -1.056722640991211], [0.5424869060516357, 1.8089286088943481, -0.4814688563346863, 0.39120879769325256, -0.6396268606185913, 0.3052960932254791, 0.748163640499115, 0.26602861285209656, -0.3380330502986908, -1.0203964710235596, -1.2153596878051758, -1.7581928968429565, 1.025707483291626, -1.2896251678466797, 0.10131789743900299, -2.2504193782806396, 0.8015449047088623, 0.4945985972881317, -0.673787534236908, -0.9020991921424866, 1.317345142364502, 0.2505493462085724, -0.6622987389564514, -0.6376365423202515, 0.49644598364830017, -0.704407274723053, 0.5739021897315979, 0.6738460659980774, -1.2723509073257446, 0.2653385102748871, -0.5395992994308472, -0.48845797777175903, 0.12371274083852768, 0.872654378414154, 0.030525216832756996, 0.36755499243736267, 0.4205463230609894, -0.9635482430458069, -0.2054203748703003, -1.3414499759674072, -0.2809055745601654, 1.638798713684082, 0.8735063672065735, -0.046033598482608795, 0.9743399024009705, -1.5075095891952515, 0.4594685733318329, -0.5303249955177307, 0.7188094258308411, -0.46112170815467834, -0.5213041305541992, -0.6825991868972778, 0.6199182271957397, 0.852247416973114, 0.892167329788208, -0.664721667766571, 0.7010008692741394, -0.002595869591459632, 0.5656251311302185, -0.38837459683418274, -0.5849373936653137, -0.7254331111907959, 0.10876654833555222, 0.7051165103912354], [-0.5528669357299805, -0.18181532621383667, 0.7301442623138428, 1.716422200202942, -1.4814934730529785, -0.6819849610328674, 0.7484449148178101, 0.023319264873862267, 0.11193645000457764, 0.7531517744064331, 0.6089040040969849, -1.6046643257141113, 0.5310443043708801, 0.15360578894615173, 0.713918924331665, 0.395504891872406, -0.3593374490737915, -0.17470823228359222, 1.0052651166915894, -0.7315395474433899, -1.1499212980270386, 0.42107364535331726, -1.2270456552505493, -0.4556550979614258, 0.059406548738479614, 0.07369984686374664, 1.5092997550964355, 1.0072213411331177, -1.4293725490570068, -1.4790465831756592, -0.7680995464324951, 0.0843454897403717, 0.4511842727661133, -0.017144832760095596, -1.9564682245254517, 0.08612751960754395, -1.735953450202942, -0.013080837205052376, -1.0493718385696411, -0.03271825239062309, -2.258025646209717, -0.8859609365463257, -1.0934202671051025, 1.1900469064712524, 0.44650739431381226, 0.07996007800102234, -0.2751213610172272, 0.34611544013023376, -1.2895115613937378, -0.7879167199134827, -0.7705356478691101, -1.0896170139312744, -1.4476553201675415, -1.599650263786316, -0.041823722422122955, 1.0683531761169434, -0.8267388939857483, 0.26099973917007446, -1.0224597454071045, -0.5665968060493469, -1.026106595993042, 0.8583504557609558, -1.0979928970336914, 0.8442544341087341], [0.11006147414445877, -1.3115864992141724, -0.37300294637680054, 0.8901634812355042, -0.21971949934959412, -1.1621860265731812, -1.5812369585037231, 0.5012996196746826, -0.5218762159347534, 0.16760973632335663, 0.45024606585502625, -1.1999958753585815, -1.0407609939575195, 0.9612408876419067, -2.436760902404785, -0.7415252923965454, 0.6546778082847595, -1.5437841415405273, -0.14478757977485657, -1.2242341041564941, 0.056994423270225525, -0.33233845233917236, 0.3371433615684509, -0.4909782111644745, -0.058429233729839325, -1.1160978078842163, 0.30393359065055847, -1.3687524795532227, 0.595395565032959, -0.5670415163040161, -0.6263175010681152, 0.17757920920848846, -0.5880081057548523, -1.7527767419815063, -0.7779827117919922, 0.831146240234375, -0.5089751482009888, 0.30556634068489075, -1.1120672225952148, -0.24841736257076263, 0.5709848999977112, 1.2542693614959717, -0.8302126526832581, 0.24465394020080566, 0.025085050612688065, 0.25038713216781616, 0.5005557537078857, -1.3575626611709595, 0.23923152685165405, 1.4767967462539673, 2.683213949203491, 1.0896432399749756, -0.15904688835144043, 0.4091600477695465, -1.415510892868042, -0.6417219042778015, 1.5943835973739624, -0.36627396941185, -1.2327587604522705, -1.1259976625442505, 0.19659650325775146, -0.5686488151550293, -0.6377893686294556, 1.0818480253219604], [0.14548495411872864, -1.1621298789978027, -0.3264721632003784, -0.7037001848220825, -0.4567604064941406, 0.03805499151349068, 0.638129472732544, 0.10788189619779587, 0.4061322808265686, -0.4119337201118469, -1.187447190284729, 0.07431116700172424, -0.43348443508148193, -0.036210253834724426, 0.114301897585392, -0.6177961230278015, -1.1083799600601196, -0.8406872749328613, -0.5325859189033508, 1.9182261228561401, 1.2511370182037354, 0.8926727175712585, 0.2600981593132019, 0.3573828339576721, -0.37198901176452637, -2.1822383403778076, -1.8635143041610718, 1.062364935874939, -0.9682785868644714, 2.3040032386779785, -0.09421468526124954, -0.8319885730743408, -0.48363906145095825, 0.3411431312561035, 1.133113980293274, -0.18039488792419434, 0.4955611824989319, -0.4093274772167206, -0.28648942708969116, 0.3789465129375458, -0.013408408500254154, 1.0833021402359009, 0.6190431714057922, -0.5223824381828308, 0.6092683672904968, -1.1786731481552124, -0.9774909615516663, 0.1857839673757553, 1.0775434970855713, 0.6277393102645874, 0.598791778087616, 1.6400580406188965, -0.031322214752435684, 0.45137152075767517, 0.8449962139129639, 0.7892549633979797, -0.5269511342048645, -0.8668287992477417, -1.1305899620056152, -0.04780995845794678, 0.5751016139984131, 0.4505116045475006, -0.7552276253700256, -0.9062016010284424], [0.8208231925964355, -0.6661306619644165, -0.16099350154399872, -0.26529932022094727, 0.11148342490196228, 1.0618972778320312, 1.9189976453781128, -0.43611034750938416, -0.3773927092552185, 0.5436729192733765, -0.4380825459957123, 0.18110869824886322, 0.5115252137184143, 0.7005249857902527, -1.4988235235214233, 0.3426955044269562, 0.3822614848613739, -1.8278685808181763, -0.9014585614204407, 0.8802434206008911, 1.5842161178588867, 0.6685030460357666, 0.42130643129348755, -0.8865863084793091, 2.0240871906280518, 0.6066861748695374, -1.2108690738677979, 0.22824378311634064, 1.302213430404663, 0.5004300475120544, 0.1538514345884323, 1.4833400249481201, 0.9425162076950073, -0.9414882063865662, 0.10958316922187805, 0.6726891994476318, 1.6087794303894043, -0.26758503913879395, 0.9833078384399414, 0.5338225364685059, -0.3282160758972168, -0.37680673599243164, -0.5093317031860352, -1.0208520889282227, -0.22581765055656433, 0.23588193953037262, -0.45335331559181213, 0.5432254672050476, 0.4929597079753876, 0.0033303683158010244, 1.3196401596069336, 2.403228998184204, -0.37736883759498596, 0.0542338602244854, -1.856437087059021, -0.4933970868587494, -0.8664162755012512, -0.9383634924888611, -0.686911940574646, 0.5964694023132324, -0.9626922607421875, 0.4218454658985138, -0.09314372390508652, -0.8623060584068298], [-1.165919542312622, -0.7781527638435364, 0.11134842783212662, 0.6323857307434082, -1.2745957374572754, -0.7326008081436157, 0.22074389457702637, 0.23187918961048126, -2.047588586807251, -0.3567844331264496, -0.5567709803581238, -0.7691925764083862, 0.40539613366127014, -0.5151458382606506, -0.785953938961029, -1.175857663154602, 0.08411213755607605, 0.2658554017543793, 2.058164596557617, 1.0602271556854248, -0.4958854913711548, 0.4008229970932007, -0.975152850151062, -0.20222395658493042, -0.763719916343689, 2.586404323577881, 0.5039812326431274, 1.7201545238494873, 0.0013843870256096125, 0.761879026889801, -0.12822405993938446, 0.3453724980354309, 1.6164615154266357, 0.1447998583316803, -3.3002214431762695, -0.2206573188304901, 0.7322946190834045, 0.7956397533416748, 0.3996204137802124, -0.37696143984794617, -1.7020751237869263, 0.3862150013446808, 0.3956996500492096, 0.6463477611541748, -1.1827826499938965, 0.32659387588500977, 0.040817998349666595, -0.3881312906742096, 0.11739012598991394, 1.106279730796814, -0.4672349989414215, -0.22847220301628113, -0.7196542620658875, -0.1296701431274414, 0.5823382139205933, 1.8308011293411255, 0.8429787755012512, -0.2521358132362366, -0.13799650967121124, 1.6779248714447021, 0.025683993473649025, -1.7764002084732056, -0.44832488894462585, 2.5009593963623047], [1.0539026260375977, 0.5103108286857605, -1.9030370712280273, -0.1380949318408966, 1.5464173555374146, -2.408933162689209, -0.14986281096935272, -0.44754335284233093, -0.6088857054710388, -1.012223720550537, -1.0932319164276123, 0.18881255388259888, 0.32136252522468567, -0.8869315385818481, 1.6541699171066284, 1.2104734182357788, -1.5012048482894897, 0.23883692920207977, 0.340163916349411, 0.8666055798530579, 1.8386858701705933, 1.0684378147125244, 0.18678653240203857, -1.4621002674102783, -0.3849544823169708, -1.9602006673812866, -0.0002280328917549923, 1.2411720752716064, -0.8314563035964966, -0.5269194841384888, 0.2684788405895233, -0.32274794578552246, 0.3882436156272888, 0.9919155240058899, 0.40466171503067017, 0.7606337666511536, -1.2869892120361328, 1.0732847452163696, 0.5860359072685242, -0.91327965259552, 0.6964730620384216, -0.38526222109794617, -0.02560417726635933, -0.10478734225034714, -1.507793664932251, 0.183314710855484, 0.17150869965553284, 0.5285865068435669, 2.2881200313568115, 0.5509120225906372, 0.6014330983161926, 0.6243225932121277, 0.3338095545768738, 2.104853868484497, 0.5745675563812256, 2.6316919326782227, -2.2764055728912354, -0.8287873864173889, -0.7607990503311157, -1.504699468612671, 0.7997365593910217, -0.21845783293247223, 0.10088158398866653, 0.6931484937667847], [-1.380606770515442, -1.2260171175003052, 0.3485552966594696, 2.457308053970337, 0.7220925092697144, -0.9719104766845703, -0.7925255298614502, 0.25143834948539734, -0.6218963861465454, 0.07599145919084549, 0.9288293123245239, 0.18574538826942444, -0.5517886281013489, 0.7161712050437927, 0.5922077894210815, 1.5077950954437256, 0.38767287135124207, 1.120929479598999, -0.4389224946498871, -0.6551036238670349, 0.4333289563655853, 0.9880568385124207, -1.1317042112350464, -0.22015374898910522, 1.3549456596374512, 0.841956377029419, -1.68895423412323, 0.2536606788635254, -0.07760647684335709, 1.374646544456482, 1.2809950113296509, 0.15404489636421204, 0.39320775866508484, -0.5185767412185669, 0.3583752512931824, -0.8660696148872375, 0.9946122765541077, -1.792653203010559, 1.669931411743164, -0.9019907116889954, 0.6128288507461548, 0.04742288589477539, 0.6213566064834595, 0.44347766041755676, -0.7596409916877747, -1.50480318069458, -0.015527489595115185, 0.7184524536132812, -0.6654689908027649, 0.04255829378962517, 1.2116249799728394, 0.5979845523834229, -0.44734859466552734, 0.20644281804561615, -0.2650613486766815, -1.0455968379974365, 1.6395512819290161, -2.0500879287719727, -0.16076336801052094, -1.238715648651123, 0.9736842513084412, 0.29258573055267334, 0.8554750084877014, -0.40210163593292236], [-1.9241818189620972, -0.13047736883163452, 0.09383974969387054, -1.6357783079147339, 1.6702860593795776, -0.13610386848449707, -1.6629756689071655, -0.06874625384807587, 0.18971091508865356, 1.0793908834457397, 1.854158878326416, 0.38173162937164307, 0.713070809841156, -0.37900471687316895, -0.26104187965393066, -0.4139033555984497, 1.6792620420455933, 0.22357873618602753, 0.3524428904056549, 0.6089113354682922, -2.47692608833313, -0.21728001534938812, 1.7186204195022583, 0.5404843091964722, -1.4370537996292114, -0.7107865810394287, 0.9340957999229431, -0.23200853168964386, 0.13340961933135986, -0.22744421660900116, -2.0630338191986084, 0.46228310465812683, 1.1395893096923828, 1.5896475315093994, -1.5585758686065674, -0.8056052923202515, -0.8326565027236938, 0.3706437349319458, 1.6792875528335571, -0.2831633687019348, -0.4725763201713562, -0.1217455342411995, 0.13021358847618103, -0.666152834892273, 1.0917569398880005, -0.6871034502983093, -0.12674003839492798, -1.0586332082748413, -0.49541017413139343, 0.013381913304328918, -0.49766892194747925, 0.8265282511711121, -0.15974847972393036, -0.42473265528678894, 0.12735888361930847, 0.8696781992912292, 0.6777863502502441, -0.3961591422557831, -0.8202043175697327, 0.3278254568576813, 0.3190818428993225, 0.5926912426948547, -0.10083860158920288, -0.5512713193893433], [-1.2075121402740479, 1.5611120462417603, -0.733802080154419, -0.11205410212278366, -0.40506529808044434, 1.0550023317337036, -1.399114966392517, -0.624736487865448, 1.0521408319473267, -0.46583589911460876, -0.0980580672621727, 0.6698544025421143, -0.06606407463550568, 0.3187987208366394, 1.3705421686172485, -2.3603665828704834, -1.5758552551269531, -0.2873951196670532, 0.367623895406723, -1.688461422920227, 1.6552801132202148, 0.026614142581820488, 0.14215873181819916, -1.1503878831863403, -0.2283944934606552, -1.035554051399231, -0.6562216281890869, -0.42178666591644287, -0.6103076934814453, -0.3537834584712982, -0.15049992501735687, -0.9022533297538757, 1.2647192478179932, 1.2216957807540894, 1.028762936592102, -0.7205950617790222, 0.31153127551078796, 0.34696459770202637, -0.4131729006767273, 1.3274290561676025, -0.6823412179946899, -2.2163679599761963, 0.09045843034982681, 0.22033463418483734, -0.13878974318504333, -0.30400747060775757, -1.6848509311676025, 0.10533256828784943, 1.553723931312561, -1.3088817596435547, -0.39905163645744324, 0.33583423495292664, -1.24168860912323, -0.44145360589027405, -0.7312831878662109, 0.6157164573669434, -0.5758079290390015, -0.3980262875556946, -1.046325445175171, 2.2066266536712646, -1.2231138944625854, 1.6771793365478516, -0.2879003882408142, -0.11357954144477844], [-0.25724247097969055, -0.7635233402252197, -0.0707472711801529, 1.262191891670227, -0.37620940804481506, 1.5446295738220215, -0.32352879643440247, -1.142669439315796, 0.16436460614204407, -0.9032636880874634, -0.1603350192308426, -0.847293496131897, -0.5541577935218811, -0.4973624050617218, 1.5914533138275146, 0.7885788679122925, -1.6654345989227295, -1.4904186725616455, 0.2579279839992523, -1.2573853731155396, -0.46547701954841614, -0.6111591458320618, -1.2532612085342407, -0.5987047553062439, 0.812109112739563, -0.5269144177436829, -0.6050105094909668, 1.55498206615448, 0.5650533437728882, 0.43235018849372864, 0.24597488343715668, -0.30495351552963257, -0.8190740942955017, -1.9612407684326172, 0.2499261200428009, 0.160318523645401, -0.6315865516662598, -0.20238392055034637, 0.6612638235092163, -2.373906373977661, -0.48014891147613525, 2.0495285987854004, 0.7352778911590576, 0.21012213826179504, -1.6022350788116455, 1.5373988151550293, 0.3579309284687042, 0.18568724393844604, 0.024078188464045525, -0.5791577696800232, -1.0699900388717651, 0.8206098675727844, 0.5459266304969788, 0.2959617078304291, 2.215642213821411, -0.400468647480011, 2.324791193008423, 0.5313499569892883, -0.5375178456306458, -0.2078939527273178, 0.5050864815711975, -0.6559998989105225, 0.5373862981796265, 0.17663778364658356], [0.1589895635843277, -0.5264178514480591, -2.956634759902954, 0.7744836211204529, -0.9762422442436218, -1.5524653196334839, 0.14655376970767975, 1.8705995082855225, -1.6423197984695435, 0.9693150520324707, -0.8497200608253479, -0.770214855670929, -0.49369245767593384, -0.4636484682559967, -1.4752992391586304, 0.08655544370412827, 0.751995325088501, 0.6401450634002686, 1.056212306022644, 1.2770427465438843, -1.0386686325073242, -1.0687265396118164, 0.5153079628944397, 1.4974603652954102, 0.796743631362915, -1.1273878812789917, -0.8298162817955017, -0.9361083507537842, -0.07352782785892487, 0.12862753868103027, -0.37676185369491577, -0.25525015592575073, -0.2563577890396118, -0.26365184783935547, -0.5797432065010071, 0.025583531707525253, -0.22030535340309143, 0.4279489815235138, 1.0541964769363403, 0.5511316657066345, 0.7590506076812744, -0.46510061621665955, 0.22054095566272736, 1.0515555143356323, -1.533611536026001, -1.5796841382980347, -0.6775268316268921, -1.32575523853302, -0.8013310432434082, -0.1586844027042389, 1.449597954750061, 0.020606035366654396, -0.9119853973388672, 0.4178542196750641, -0.45499473810195923, -0.05983823165297508, -0.31576770544052124, 0.5269801616668701, 0.10578080266714096, 0.26314184069633484, 0.032027408480644226, 0.8373838663101196, -0.4282102882862091, -0.5305525660514832], [-1.2029613256454468, 0.683458685874939, 0.17949262261390686, -1.1929395198822021, -0.023618727922439575, -0.08984005451202393, 0.7787250280380249, 0.7369359731674194, -0.8074346780776978, -1.0028138160705566, -0.4679301679134369, 0.4763842821121216, 0.15503978729248047, 0.8277096152305603, -1.6721031665802002, -0.5998677611351013, 2.4439597129821777, -1.273245096206665, -1.0282975435256958, -0.8629514575004578, -0.8873261213302612, 0.23929175734519958, -0.12348411232233047, 0.05501434579491615, 0.855746865272522, 0.035800378769636154, 0.6642447113990784, 0.9399405717849731, -0.4937310218811035, 0.9235047101974487, 1.0484343767166138, 0.9574037194252014, -0.25997358560562134, 0.9294436573982239, 0.023778598755598068, -0.7948402762413025, 1.1171854734420776, 0.4903630316257477, 0.3340487778186798, 1.04648756980896, -0.6865678429603577, -0.2788654863834381, 0.20615403354167938, 0.07470575720071793, -1.7294892072677612, 0.6767372488975525, 2.2116353511810303, -0.32840579748153687, -1.793325424194336, 0.29680103063583374, -2.3644590377807617, -2.103431463241577, 1.7131013870239258, 0.7717457413673401, 1.610174536705017, -0.43071553111076355, -0.5673263072967529, -0.9870213270187378, -0.9082636833190918, 0.019048474729061127, 0.31865453720092773, -0.2612365782260895, -1.7312464714050293, 0.4101904332637787], [0.8816173076629639, -1.8693095445632935, -0.742178738117218, -0.5889154076576233, 1.0842570066452026, -1.392795205116272, 1.9222418069839478, 0.1415730118751526, 0.802504301071167, 0.3155086636543274, 0.8491958379745483, 0.2241504192352295, -1.3448195457458496, 0.48641520738601685, 0.020529281347990036, 1.9211390018463135, -0.49796658754348755, -1.2261990308761597, 0.16336104273796082, 0.24096250534057617, 0.28850579261779785, -0.5738539099693298, -1.6931111812591553, 0.23063711822032928, 0.475102961063385, 0.6827939748764038, -0.610812246799469, 0.15769609808921814, 1.1950769424438477, 0.09722281247377396, 1.398468255996704, -1.335591197013855, 0.1809946596622467, -1.5377204418182373, 0.11999496817588806, -1.2696267366409302, 0.9060244560241699, 0.2356429100036621, -1.4600640535354614, -0.13701745867729187, -0.39240771532058716, -1.6896193027496338, -0.4643528461456299, 0.1840902864933014, 0.7879387736320496, 0.27756133675575256, 0.3486253023147583, -0.2824341356754303, -0.6554558873176575, 0.6839361786842346, -0.7013142108917236, 1.399699091911316, 0.47495001554489136, 0.44487011432647705, 2.329505681991577, -0.09214973449707031, -0.28823623061180115, -0.4995911717414856, -0.3868809640407562, -0.06727619469165802, -0.2890443503856659, -0.8744402527809143, -0.43641141057014465, 0.30803078413009644], [-0.6257743835449219, -0.18713659048080444, -0.28468793630599976, 0.5499782562255859, -0.3028746247291565, -0.33157187700271606, -0.07839307188987732, 0.595453679561615, -0.5776413083076477, -0.9053755402565002, 0.5346114635467529, 1.1852071285247803, -1.5219128131866455, -0.6053091287612915, 1.2092565298080444, -0.2004365622997284, 0.0922105610370636, -0.8408598303794861, 0.1312190592288971, -1.1078224182128906, -0.8481083512306213, -0.51041179895401, 1.1388180255889893, 0.4492916762828827, 0.1505206823348999, -0.19693370163440704, -1.376981496810913, 1.2499897480010986, -0.9898215532302856, 0.39507102966308594, 1.7904691696166992, 0.000644169223960489, 1.3153997659683228, 0.009924421086907387, -0.6246404647827148, 0.6138252019882202, 0.4080589711666107, -1.2227345705032349, -1.3248982429504395, -0.34617167711257935, -2.256075382232666, 0.05458030104637146, -0.22654543817043304, -0.7170400619506836, -2.068603992462158, -0.049630917608737946, -1.4317312240600586, 0.003260689089074731, 0.40838560461997986, 0.6370511651039124, -0.033901337534189224, 1.852907419204712, -1.7327704429626465, -0.4944130778312683, -0.7323442697525024, 2.087611436843872, -0.6537451148033142, 0.3477664589881897, -0.1526053249835968, -0.2858375608921051, 1.7666999101638794, 0.068528912961483, 0.004294941667467356, -0.14553357660770416], [0.39467188715934753, -0.9473170042037964, -0.22066952288150787, -1.178117036819458, 0.1408330351114273, -0.07882342487573624, 0.052821505814790726, 1.603016972541809, -0.12887293100357056, 0.32791978120803833, 1.0564854145050049, 1.45151686668396, 0.2111135572195053, 1.400294303894043, 0.16209736466407776, 1.2502431869506836, 1.148133397102356, -1.5903007984161377, -0.5260705351829529, 0.39740538597106934, 1.2244088649749756, 0.30425894260406494, -0.2625449597835541, -0.5414348840713501, 1.3540300130844116, 0.7184270024299622, 0.7622726559638977, 0.12799043953418732, 0.9932078123092651, 2.1675188541412354, 0.03904865309596062, 1.8964331150054932, 0.9296445846557617, 0.3929128646850586, 0.2872474491596222, 1.6569207906723022, -1.0401725769042969, -0.3503178060054779, -0.2752101421356201, 1.0665088891983032, 1.0238189697265625, -0.2177448272705078, 0.26321786642074585, 0.4137955605983734, 0.1303492784500122, 0.04204424098134041, 0.6498411893844604, 0.8390375375747681, 1.066004753112793, -0.5118035078048706, -0.8527626395225525, -1.0238237380981445, -1.0061856508255005, 0.7822340130805969, -1.1072864532470703, -0.6025813817977905, 1.0937330722808838, 0.6672469973564148, 0.022114410996437073, 0.9156147837638855, -1.1429067850112915, -1.2412461042404175, -1.1061407327651978, 0.4790208637714386], [0.6758094429969788, -0.15251238644123077, 0.05454602837562561, 0.34759968519210815, 0.6166449785232544, -1.0381356477737427, -0.07330811768770218, 0.3087676763534546, -0.4331766664981842, 1.5036805868148804, -0.14470238983631134, 0.2745918333530426, -0.746975839138031, -1.1521999835968018, 0.7063512206077576, 0.12799586355686188, 0.19829601049423218, -1.713633418083191, -0.1982416957616806, -0.03033505566418171, -0.45835670828819275, -0.8522163033485413, -0.7890672087669373, -0.22561821341514587, -0.6098114252090454, 1.6142112016677856, -0.7360857129096985, -0.6596062779426575, 0.943800151348114, -1.2780929803848267, 1.059544563293457, 1.1439157724380493, -0.6186276078224182, -0.2049701064825058, 0.20358338952064514, 0.3689923584461212, 1.0806691646575928, 0.8643887042999268, 1.5404613018035889, -1.0254477262496948, 0.0233056228607893, 0.8848638534545898, 0.7277822494506836, -1.0759400129318237, 0.6559069156646729, -0.5490737557411194, -0.8523083329200745, 2.1584854125976562, -1.3293718099594116, -1.0713027715682983, 0.04399770498275757, -0.9667739272117615, -0.429703027009964, -0.32216718792915344, 0.01909838803112507, -0.6240717172622681, -1.0791693925857544, 0.20901423692703247, 0.591572642326355, 1.4379642009735107, -0.10142611712217331, -0.31800517439842224, 0.2613573372364044, -1.2389622926712036], [0.4155258238315582, 0.149495929479599, -0.23314274847507477, -0.41840410232543945, -1.1597702503204346, 0.045765385031700134, 0.48373666405677795, -0.05722067505121231, 0.6646448969841003, 1.092580795288086, -0.05152151733636856, 0.8357780575752258, -0.6013839244842529, -0.8110374808311462, -0.32526126503944397, -1.2313082218170166, -0.43448761105537415, 1.2783442735671997, -0.37677568197250366, 0.6552053689956665, -0.7234767079353333, 0.6301261186599731, 0.35712847113609314, -0.7242764234542847, -0.6954771280288696, 2.6178667545318604, 0.661130428314209, 0.6129764318466187, -2.104314088821411, 0.6692700982093811, 0.7380174994468689, -0.3208307921886444, 1.041165828704834, 0.6295158267021179, 1.1804263591766357, -0.9687676429748535, 0.3377493917942047, -1.823913335800171, 0.53171306848526, -0.0013706512982025743, 0.6104368567466736, -0.5903968214988708, 0.1277582347393036, -0.7767245769500732, 0.7126196622848511, 0.7446787357330322, 1.3381754159927368, -0.8667832016944885, 0.6843321323394775, -0.5679384469985962, 0.49789053201675415, 0.4750714898109436, 2.4017584323883057, 0.45381975173950195, 1.0563280582427979, -1.0159441232681274, -0.18532727658748627, -1.0144492387771606, -0.035288795828819275, 0.9768315553665161, 0.03257141262292862, -0.9625467658042908, -0.22117699682712555, 0.06987152993679047], [2.1282613277435303, -2.2278406620025635, -1.442710041999817, 0.498312771320343, -0.9828256964683533, 0.7362269759178162, -1.772687315940857, 0.7835094928741455, -0.7350298762321472, -1.5934967994689941, 0.40592607855796814, -1.2109686136245728, -0.692084789276123, 0.2636469900608063, 0.5336755514144897, -0.5201438665390015, 1.5948283672332764, 0.5750368237495422, 1.2132010459899902, -0.9388840794563293, 0.6145938038825989, -1.289775013923645, -0.4481736123561859, -0.6466748118400574, -1.3339638710021973, 0.08122479170560837, -1.5318578481674194, -2.31290602684021, -1.3533761501312256, 0.4425535202026367, 1.9252917766571045, -1.258973479270935, -1.4864907264709473, 0.29063382744789124, 0.941679060459137, -0.2942352294921875, -0.287374347448349, 0.7509495615959167, 1.8292758464813232, -0.06291963160037994, 0.35749784111976624, 1.0725762844085693, 0.6500111818313599, -1.2118631601333618, -0.7580516934394836, 0.19076523184776306, 2.1941936016082764, -0.37407973408699036, 0.06401938199996948, 0.2633933424949646, -0.057880714535713196, 1.1419310569763184, 0.06772830337285995, -1.2957568168640137, -0.01621245965361595, -0.655964195728302, -0.14426586031913757, -0.9648016691207886, -0.48852968215942383, 0.1854320466518402, -1.322016954421997, 0.5345818996429443, 0.8814384341239929, -0.2998819351196289], [0.11235689371824265, 0.6065040826797485, 1.0984387397766113, -0.4047507643699646, -0.7537661194801331, 0.1061035767197609, 0.26758772134780884, -0.8024374842643738, -0.507203221321106, 1.3785405158996582, -0.07551422715187073, 0.04690317437052727, 0.5006139278411865, 0.1205083504319191, 1.7946850061416626, -1.2774730920791626, 1.749491572380066, -0.7590218186378479, -0.8513619303703308, 2.2352519035339355, -1.2857179641723633, 1.5211068391799927, 0.8330625295639038, -1.7051730155944824, 1.2237789630889893, 0.09770091623067856, 1.7963069677352905, 0.655175507068634, 0.7281693816184998, -1.4390928745269775, -0.004378599114716053, 1.0050934553146362, 0.17271843552589417, -1.6002994775772095, 0.8155708909034729, -0.7236570715904236, 1.1354583501815796, -2.348784923553467, 1.268694281578064, 0.18000644445419312, -1.0691665410995483, 1.0047677755355835, -1.7513558864593506, -0.743679404258728, -0.19864605367183685, -0.6405066847801208, 0.511961817741394, -0.4865025281906128, 1.252233862876892, 0.11354462802410126, -0.027620501816272736, 0.02964680641889572, 0.11615351587533951, -1.27675461769104, 0.2644963562488556, -0.12670773267745972, 1.5753921270370483, -0.5039933323860168, -0.9216621518135071, -0.7950071096420288, 0.9409099221229553, -0.6205770969390869, -0.7996944189071655, -0.2833089530467987], [-1.3165706396102905, 1.3261723518371582, 1.031518816947937, -1.7395309209823608, 1.9642540216445923, -0.5655879378318787, -1.398568868637085, 0.5667205452919006, 1.7934284210205078, 0.43408632278442383, -0.23151186108589172, 0.38077011704444885, 2.695958375930786, 0.2266167551279068, 0.6956424713134766, -1.0598536729812622, 0.8355875015258789, -0.4483480155467987, 0.3550948202610016, -1.0459955930709839, -1.4780980348587036, -0.5123730897903442, 0.36676982045173645, -0.18526266515254974, -1.702608585357666, 0.6626041531562805, -0.7350330948829651, -1.0201001167297363, 0.5523480772972107, -0.07070472836494446, -0.9189363121986389, -1.0116790533065796, 0.5616627335548401, -0.28439652919769287, -2.3760902881622314, -2.7554244995117188, 0.1528881937265396, 0.28369376063346863, 1.0841355323791504, 1.727997899055481, -0.8748651146888733, 0.9743981957435608, 1.0013898611068726, 1.936239242553711, 0.9650405049324036, 0.6575071811676025, -3.0217697620391846, 1.4926658868789673, -0.10305953770875931, -2.6229076385498047, -0.49155914783477783, -1.431121826171875, 0.18994921445846558, 0.026531409472227097, -1.4661555290222168, 0.042217645794153214, -1.531686544418335, 0.2821388840675354, -0.6819878220558167, -0.535326361656189, -0.4986175000667572, -1.461972951889038, -0.013634191825985909, -0.1584056317806244], [-0.5373010039329529, -0.18919861316680908, 1.097988486289978, 0.8586548566818237, -0.42364296317100525, 1.5762839317321777, -1.170066237449646, 0.924404501914978, -1.3059216737747192, -0.22430288791656494, -0.19726210832595825, -0.13066993653774261, -1.079708456993103, -0.3595511019229889, 0.7043192386627197, 0.20256654918193817, -0.42068567872047424, -1.1865217685699463, 0.16487811505794525, -0.21522991359233856, 0.5094784498214722, -1.8299895524978638, -0.2448946088552475, -0.9055017232894897, 0.3614587187767029, -0.7852491140365601, 0.08995581418275833, 0.37485772371292114, 1.5489641427993774, -1.7065762281417847, -0.09647087007761002, 2.0968456268310547, 0.3128836750984192, -0.13847902417182922, -0.7447444200515747, -0.7139413356781006, 2.001004695892334, -0.7019301056861877, 1.008933186531067, 1.47663414478302, 0.20367759466171265, -0.4535675048828125, 0.0857599675655365, 0.681178867816925, -1.3242892026901245, -1.572842001914978, 0.9915528893470764, -1.337689757347107, -1.2259116172790527, 0.38387781381607056, 1.2456401586532593, 0.44173693656921387, -0.43800413608551025, -1.4809296131134033, -0.24858517944812775, -0.12313719093799591, -0.7132056355476379, -0.8770250082015991, 0.09207342565059662, 0.42191869020462036, -0.24457572400569916, 3.267598867416382, 1.1369789838790894, 1.1017777919769287], [-2.6897950172424316, 1.683335781097412, 0.13235558569431305, -2.109901189804077, 0.448473185300827, 0.38192474842071533, 1.5664013624191284, 0.9873456358909607, -0.2171122133731842, 0.12725716829299927, -1.1823915243148804, 1.9300273656845093, -0.31530699133872986, 1.1086540222167969, -1.1810986995697021, 0.1335691213607788, -2.215620279312134, -1.563305377960205, -1.282668948173523, 0.8440549969673157, -2.513334274291992, -0.3017089068889618, 0.23624099791049957, 1.4949191808700562, 0.4813280701637268, -1.930614709854126, -0.175009623169899, 1.867447853088379, 0.3721565902233124, -0.03646562620997429, 1.168684720993042, 0.049158304929733276, 0.5304222702980042, 0.2581089437007904, -0.60979825258255, 0.27431726455688477, 1.1259489059448242, 1.6090284585952759, 1.6599477529525757, -0.2216477245092392, -0.14854344725608826, -1.0534719228744507, -0.5669771432876587, -0.2560315430164337, 1.646586537361145, 0.963152289390564, 0.47415849566459656, -0.8853880167007446, 0.8379145860671997, -0.5135824084281921, 0.014619264751672745, -1.6909515857696533, 0.9074704051017761, -1.2210613489151, 0.5188750624656677, -1.6211323738098145, 0.5429428219795227, 1.6327159404754639, 1.499893307685852, -0.24964144825935364, -0.1294752061367035, 0.35406026244163513, 0.7655460238456726, 0.6391651034355164], [0.5093585848808289, 1.4852519035339355, 0.23420828580856323, 1.751572608947754, -0.32510673999786377, -1.3681780099868774, -0.9732339382171631, -0.176749587059021, -1.6448698043823242, -0.33851441740989685, 1.1830922365188599, -0.63978511095047, -1.412009596824646, 0.06323976069688797, -0.22870886325836182, -0.4103213846683502, -0.04816780239343643, 1.2223659753799438, 0.5745480060577393, -0.5062878131866455, -0.36660563945770264, -0.07536562532186508, -1.7526488304138184, 1.412933111190796, 0.5028767585754395, -0.3149392604827881, 0.19419750571250916, -0.46512719988822937, 0.5737331509590149, -0.48427262902259827, 0.25438427925109863, -0.10462636500597, -0.46592605113983154, -0.4979862570762634, 1.0155760049819946, -0.20760734379291534, 0.22891749441623688, 0.17820224165916443, 0.5507968068122864, -3.175855875015259, 0.03088895045220852, -0.962993323802948, -1.7035655975341797, 0.5808220505714417, -1.3695745468139648, -0.34606608748435974, 0.8675856590270996, 0.03141477704048157, 0.30930575728416443, -2.249728202819824, -0.7232149243354797, -0.5152484774589539, 0.8805435299873352, 1.8898817300796509, 0.9225802421569824, -1.757798433303833, -2.2870471477508545, 0.8875118494033813, -0.09951745718717575, -0.21480576694011688, 0.44684603810310364, -1.011852741241455, 0.25499191880226135, 1.9865658283233643], [0.6177988052368164, -1.279312014579773, -0.2677137851715088, -1.3188246488571167, 0.5910429358482361, 0.733853280544281, -0.28870511054992676, 0.8668851852416992, -1.6583300828933716, -0.2841394245624542, -0.5648565292358398, 0.04953225702047348, -0.7680376768112183, 0.8051901459693909, -0.7615500688552856, -0.2923467457294464, 0.7901478409767151, 0.09786492586135864, 0.6466538310050964, 0.9448859095573425, 1.0061196088790894, 0.06932958960533142, -0.4578847885131836, -1.6846169233322144, 0.7079563140869141, 1.7141433954238892, 0.22758446633815765, 0.5911744832992554, 0.061956021934747696, -0.933707058429718, -0.2507001459598541, -1.50739324092865, 0.6508116722106934, 0.901199460029602, -0.47588911652565, 0.06616926938295364, 1.0977200269699097, 0.696625828742981, -0.3109581768512726, 0.6476309299468994, -0.3341836631298065, 0.16493310034275055, 0.18333443999290466, 0.8441798090934753, -0.760715901851654, 0.4977589547634125, 1.1845611333847046, 0.1497686207294464, 1.0123722553253174, -0.4936511516571045, 0.7161370515823364, 0.07544811815023422, -0.08094511926174164, 0.8753316402435303, 0.9261494278907776, 0.2752452492713928, -1.8092799186706543, 0.6778146624565125, -1.8219943046569824, -0.19062718749046326, 0.7048513293266296, 0.7635765671730042, -1.4683637619018555, 0.39173218607902527], [0.2441118061542511, -1.8516160249710083, 0.04544977471232414, -0.006445911712944508, 0.015927284955978394, -0.032061148434877396, 0.3987214267253876, 1.3745485544204712, -2.2150566577911377, -1.1488767862319946, -0.04873903468251228, 0.07813259214162827, 0.3697088360786438, -1.1978355646133423, 0.49703720211982727, -0.7720741629600525, 0.45651862025260925, 1.5250327587127686, 0.06121246889233589, -0.37496045231819153, 0.8320062160491943, 0.46945029497146606, 0.32942911982536316, 0.9745515584945679, -0.8336819410324097, -0.8418322205543518, -2.3580241203308105, 1.0772850513458252, -0.37213167548179626, -0.39350661635398865, 0.35713285207748413, -0.7900114059448242, 1.6162713766098022, -0.4344172775745392, -0.4699663519859314, 0.9076529741287231, 0.5574879050254822, 0.6164868474006653, -0.14662045240402222, 0.6920430064201355, -0.50490403175354, 0.9030390381813049, -0.5977894067764282, -0.5373605489730835, -0.4540534019470215, 0.5952143669128418, 0.9643673300743103, -1.4648141860961914, -1.0057356357574463, -0.6304657459259033, -0.6870419383049011, 0.5566553473472595, 0.6537222862243652, 1.4760102033615112, 0.9630969166755676, 0.9998336434364319, -0.9413207769393921, -1.077332854270935, 0.9464991688728333, -0.848602831363678, -0.07018734514713287, 0.5979719758033752, 0.7083553075790405, -0.46044060587882996], [0.4552285373210907, -2.275902271270752, 1.19141685962677, 0.27228105068206787, -0.8108251094818115, -0.6597021222114563, -0.5722678899765015, 0.22684352099895477, 0.346240758895874, 0.4577898383140564, -1.0525028705596924, 1.1748491525650024, 0.3612678647041321, 0.9490302205085754, -0.5167796015739441, 0.725758969783783, 0.5173929333686829, -0.31035953760147095, 0.2814269959926605, 1.7495442628860474, -0.1625368595123291, -1.3188427686691284, -0.004712255205959082, 0.7267642617225647, 0.6193611025810242, -0.502447783946991, 0.3944067060947418, -1.364819049835205, 1.2056214809417725, 1.4239436388015747, -1.138719081878662, 0.8162877559661865, 0.6192126870155334, 0.7079226970672607, -0.4291335940361023, 0.24936869740486145, 0.26479843258857727, 0.9031575322151184, 1.1861743927001953, -0.19612793624401093, -1.5034409761428833, -0.7139133214950562, -0.24035508930683136, 0.1879516988992691, 0.4425918757915497, -1.3331434726715088, 0.3130311667919159, 0.5395607352256775, -0.23679326474666595, -1.871375322341919, 1.2974705696105957, -0.24233530461788177, 0.9924355745315552, 0.08291709423065186, -0.13549074530601501, 0.15484553575515747, 1.3683031797409058, -0.41373515129089355, -0.6066861152648926, 0.7680736780166626, 0.9538628458976746, 1.278519630432129, -0.2528059780597687, -0.4355180859565735], [-1.100740671157837, 0.5426524877548218, 0.10325994342565536, -0.6161899566650391, 0.12068583816289902, 0.5645413994789124, 1.1319372653961182, 0.31530898809432983, 0.44556164741516113, 0.353866845369339, 0.5030397772789001, 1.2520768642425537, -0.4256448745727539, -0.2798347771167755, 2.3651392459869385, 0.04059962183237076, 0.1730417162179947, -2.143845796585083, 1.0444530248641968, -0.025197384878993034, 1.4787945747375488, 0.7382291555404663, 0.33751723170280457, -1.177311658859253, 0.7618849277496338, 1.7118779420852661, 0.153799906373024, -0.3534589111804962, 2.18131685256958, 0.23665578663349152, -0.20602165162563324, 0.16212230920791626, 0.8841331005096436, -0.6576430201530457, -0.7195529937744141, -1.7792794704437256, -0.945466160774231, 0.6151774525642395, 0.17360180616378784, -0.07158626616001129, -0.40655022859573364, -1.0271966457366943, 2.099280834197998, 0.9160158038139343, 0.08728877454996109, 0.780614972114563, 0.5389388799667358, 0.4517644941806793, 1.7591865062713623, 0.3389332890510559, 0.08467621356248856, 0.4078201353549957, -0.17435172200202942, -0.6410276889801025, -0.4528637230396271, 1.7257550954818726, -1.2444756031036377, 0.08126796036958694, 0.01849552057683468, 0.3842437267303467, -0.5328895449638367, -0.10464435070753098, 0.6876689791679382, -0.6731180548667908], [-0.10667158663272858, 2.6398637294769287, -1.53402841091156, -0.6632573008537292, 0.2586708962917328, 0.43194806575775146, 0.5024601817131042, -1.6395624876022339, -1.2560008764266968, -0.029406672343611717, 1.4466817378997803, -1.3775312900543213, -0.2580062448978424, 1.300350308418274, -0.03621204197406769, 0.7554560303688049, 1.595215082168579, 1.1105459928512573, 0.39037901163101196, -0.23419618606567383, -1.1083177328109741, -2.2469911575317383, -0.5905048251152039, 0.6897148489952087, 0.3329818844795227, -0.5604501366615295, -0.7206988334655762, 0.6082789897918701, -1.8181442022323608, 1.8637641668319702, 0.5747593641281128, 0.17793352901935577, 0.6403735280036926, 0.423630952835083, 0.24036498367786407, 0.4890731871128082, 2.149902105331421, -0.7030695676803589, 0.6766260266304016, -2.446240186691284, 0.06490190327167511, 0.33561527729034424, 0.3369157314300537, -0.3227123022079468, 0.6997547149658203, -1.6751620769500732, 0.44600269198417664, 0.6627655625343323, 1.4143601655960083, 0.13859817385673523, 0.4111538231372833, 0.37192705273628235, -1.4165549278259277, -0.6794725656509399, 0.5085604190826416, 3.403900623321533, -0.3355368971824646, -0.5288885831832886, -0.2687440514564514, 0.49441075325012207, -0.856968879699707, 1.274458408355713, 0.021754730492830276, -0.357890784740448], [-0.09023894369602203, -0.23653167486190796, 1.1180938482284546, 1.0879511833190918, -1.1185081005096436, -0.9036962389945984, -0.7024897933006287, 0.7210291028022766, -0.38520556688308716, 0.3644983172416687, -1.4504328966140747, -0.034444887191057205, 1.3480817079544067, -0.3102400600910187, -0.3096308410167694, -0.7359599471092224, -0.22176137566566467, -1.1824346780776978, 1.5873351097106934, 0.6770404577255249, -2.577707529067993, 0.4662693738937378, -1.6196904182434082, -0.9568107724189758, 1.6901568174362183, 0.5131210088729858, 0.7451138496398926, 0.008779416792094707, 1.7508240938186646, 0.7951120734214783, 0.29932379722595215, -1.3524765968322754, 0.9287505149841309, 1.1762739419937134, 0.12634322047233582, 0.09661024808883667, -2.1642117500305176, -0.3885836899280548, 2.351985454559326, 0.3648226857185364, -1.7745940685272217, -0.7293849587440491, 1.12441885471344, -0.8557504415512085, 0.1597999781370163, -0.8339307904243469, -0.8391550779342651, 1.1711091995239258, -1.2322818040847778, 0.7633906006813049, -0.3501209020614624, -0.7056121230125427, 1.1268441677093506, -1.794937252998352, 0.8566466569900513, 0.35298043489456177, -0.6316512823104858, 0.564895510673523, 0.19442817568778992, -1.6512089967727661, 0.9915313720703125, 0.5824700593948364, 0.6757727861404419, -2.7416927814483643], [0.3480677604675293, -0.17039835453033447, 1.0585370063781738, 0.8300047516822815, 1.5646815299987793, -0.2886895537376404, -1.3474944829940796, 0.3263693153858185, -0.26204681396484375, 0.21242602169513702, 0.2901689410209656, 1.4487160444259644, 1.6140291690826416, -0.6273233294487, -2.0359342098236084, -1.8354867696762085, 0.477038711309433, -0.8334892988204956, 0.9240762591362, -1.8234679698944092, -1.6808948516845703, -0.027107100933790207, 0.3089691400527954, -1.5633299350738525, 0.8584248423576355, -0.5014059543609619, 2.022765874862671, 1.3906446695327759, 0.5852870345115662, 0.05891185626387596, 1.8488447666168213, 0.34368181228637695, -0.6323792934417725, -0.5996883511543274, 0.40514734387397766, 0.2882404923439026, -0.9710066318511963, 0.57342129945755, -1.0230019092559814, -0.07965218275785446, -1.0777649879455566, -0.24021796882152557, -1.9987307786941528, 0.3233012855052948, 0.5418450236320496, 0.9644614458084106, -0.9315747022628784, 0.08427341282367706, -0.7659024000167847, -0.3632560074329376, 0.02954500913619995, 1.2171282768249512, 0.1475679874420166, -0.5199012160301208, 0.5295628905296326, -1.4442776441574097, -0.6500863432884216, -0.40791693329811096, 1.3612565994262695, 1.2970527410507202, -1.3492040634155273, -0.5538473129272461, 0.4060826897621155, 0.42656615376472473], [-0.41924232244491577, 0.36247825622558594, 1.1155164241790771, -1.6188114881515503, -0.21275049448013306, 0.3473886251449585, 1.4563367366790771, -1.0570954084396362, 0.6259441375732422, -0.08622431010007858, 0.32703229784965515, -0.7105135321617126, -0.058529093861579895, -0.46562832593917847, -1.7288262844085693, -0.2606937289237976, -0.8280887603759766, 0.508530855178833, 0.44811806082725525, -2.24131178855896, -1.9970388412475586, 0.833675742149353, -1.2591153383255005, -0.14114753901958466, 0.1307152509689331, 0.9082502126693726, 1.2842347621917725, -0.13802380859851837, -0.10735735297203064, -1.7341793775558472, 0.5244992971420288, -0.08917957544326782, -0.8694059252738953, -1.571485161781311, -0.6755408048629761, 1.4071649312973022, 1.138317346572876, -0.47351014614105225, -0.09299934655427933, -0.6728001236915588, 1.3028690814971924, -0.36187076568603516, 0.45694801211357117, -1.5589861869812012, -0.8215099573135376, -1.4598380327224731, -0.6173429489135742, -0.519101083278656, -0.08265382796525955, 0.4683583080768585, 0.314702570438385, -0.989075243473053, 0.8199536800384521, -0.8930916786193848, 0.8773518204689026, -1.284576416015625, -0.20300590991973877, -0.07392435520887375, 2.2896170616149902, -0.6960369348526001, 0.19897043704986572, 0.47608208656311035, 1.2094957828521729, -0.14743581414222717], [1.009353756904602, -0.4662638306617737, -0.10506675392389297, 0.040625546127557755, 0.3322595953941345, -0.04841900244355202, 1.857784390449524, 1.6745775938034058, -1.6327006816864014, 0.43402448296546936, 0.7706924676895142, -0.3572676181793213, -0.872446596622467, -0.01672092080116272, 2.3478715419769287, 0.17689956724643707, -0.6453829407691956, -1.9994010925292969, 2.952116012573242, -0.40749502182006836, 0.17623014748096466, 0.18447640538215637, 0.9993001222610474, 0.349465012550354, 0.5392512679100037, -0.8832855224609375, -0.8145463466644287, -0.4650944173336029, 2.7397098541259766, 0.35008567571640015, 0.3067369759082794, 1.4730945825576782, 1.0510857105255127, 1.4403449296951294, 0.5472983121871948, -0.8091573119163513, -0.5282605886459351, 1.345168113708496, -0.6090695261955261, -0.2579309940338135, -1.0146533250808716, -1.2837992906570435, -2.668612480163574, 1.1890078783035278, 0.08656207472085953, -1.3773987293243408, 0.19697068631649017, 0.953823983669281, 0.8763507604598999, -0.045293282717466354, 0.9466695189476013, 0.24624323844909668, -0.42834535241127014, -1.6972718238830566, -0.4811389446258545, 1.1251652240753174, 0.2115008533000946, -0.9449208974838257, -1.4774545431137085, 0.2157943844795227, 0.4758153557777405, -1.4204537868499756, -0.7275373339653015, -0.15611378848552704], [0.15668287873268127, -1.1778969764709473, -0.8372963070869446, -0.32560354471206665, 0.453675776720047, -0.290284126996994, 0.6085909008979797, -0.7987721562385559, -0.14195723831653595, 1.081700325012207, 1.7855947017669678, -0.4820219576358795, -0.499607652425766, -1.178610920906067, 1.2547149658203125, -1.3243699073791504, -1.1030404567718506, 1.558998942375183, -0.5111984610557556, -0.060609519481658936, -0.36747175455093384, 1.001660943031311, 0.12316206842660904, 0.6724134683609009, 0.25727149844169617, 0.28690260648727417, 0.7357181906700134, 1.8513915538787842, -1.3476771116256714, 0.9673908352851868, -0.7548069357872009, 1.0579030513763428, 0.6765890121459961, 0.5701075196266174, 0.6358225345611572, -0.6686615943908691, -0.25872382521629333, -0.24296477437019348, -0.03410409763455391, 0.2548307180404663, 1.3519541025161743, 0.8854088187217712, -1.2768300771713257, 0.32556208968162537, -0.38540762662887573, -0.10171359777450562, 0.8778790831565857, -0.7297895550727844, -1.4578858613967896, 0.3566349446773529, 0.20002642273902893, 0.7564961910247803, -0.9797113537788391, 0.3754878640174866, -0.3416954278945923, -0.3072924315929413, 1.3996027708053589, -0.6581388115882874, 2.177664279937744, -1.3563141822814941, -1.2463254928588867, 0.8301738500595093, 0.010742027312517166, 1.0641727447509766], [-1.6697096824645996, -1.0867514610290527, -0.7128974199295044, 1.382501482963562, -0.05124601721763611, 0.5598719120025635, 0.33540675044059753, 0.8960771560668945, 1.9531749486923218, 0.13554948568344116, -1.1300742626190186, 0.5993415713310242, -0.5238505601882935, -1.2143646478652954, 1.4962375164031982, -1.006061315536499, 1.0813324451446533, 0.5938038229942322, 1.0654758214950562, -0.4401884973049164, -1.346893310546875, -0.12362585216760635, 1.1478204727172852, -1.8257707357406616, -0.02241959609091282, 0.36522185802459717, -1.3019165992736816, 0.2071150690317154, 0.9771959781646729, -0.03238454461097717, 0.01825689896941185, 0.029311902821063995, -1.3711268901824951, 0.9255171418190002, 1.7218629121780396, 0.8050751686096191, -0.07298900187015533, 2.9905173778533936, 0.7952788472175598, -0.3487798571586609, -0.7603176236152649, 1.065003752708435, -0.7205275893211365, -1.018066167831421, -0.26412150263786316, 1.0272064208984375, 1.0588663816452026, -1.1816565990447998, -1.530666708946228, 0.0021830990444868803, -1.4385688304901123, 0.7830008268356323, -0.38907569646835327, 1.7857416868209839, 0.3888024687767029, 1.107825756072998, -1.386195421218872, -0.3693082630634308, 1.4557164907455444, 0.5472659468650818, 1.0925984382629395, -1.000240683555603, 0.12847012281417847, -0.2357005774974823], [-2.2282283306121826, -0.9493234753608704, 0.9458318948745728, 0.48949772119522095, 0.28347885608673096, -0.6300256848335266, -0.6333368420600891, 0.8927921056747437, -1.7646080255508423, -0.6409952640533447, -0.6530243158340454, 1.364155650138855, -0.6277003288269043, 0.8386269807815552, -0.39930835366249084, -0.4987763464450836, 0.11393687129020691, 1.4007971286773682, -1.2883615493774414, 0.8950769901275635, 0.8141374588012695, 0.9707729816436768, -0.5455570816993713, 1.0951848030090332, 0.3137815296649933, 1.5998598337173462, -0.608142077922821, -0.04818277060985565, -0.015571468509733677, 1.0345085859298706, 2.2875633239746094, 0.09386079758405685, 0.07055163383483887, -1.2849041223526, 0.9417647123336792, -2.22510027885437, 0.11217644065618515, 0.4964105784893036, 0.20171457529067993, 0.1912577748298645, -0.08767925947904587, -1.3925751447677612, -1.6126364469528198, 0.22249048948287964, 0.8863314390182495, 1.6828715801239014, 0.211982861161232, -0.247919499874115, -0.10709960013628006, 1.698968529701233, -0.7646144032478333, -0.8482460379600525, -1.0209472179412842, 0.18604932725429535, 0.31048303842544556, 1.3738503456115723, -0.8777323365211487, 0.13961437344551086, -0.3404846787452698, -1.2109981775283813, -0.13399724662303925, 0.022269539535045624, -1.047049880027771, 0.1784128099679947], [-1.2456319332122803, 0.41262003779411316, 1.0555028915405273, 3.4227352142333984, 0.5953267812728882, 0.6716184616088867, 0.16797547042369843, -0.3317871689796448, 1.6676173210144043, 0.23910459876060486, 0.8960509896278381, 0.1769842654466629, 0.7010977268218994, 1.476542592048645, -0.8927711844444275, 1.1594171524047852, 0.10340354591608047, -1.3120057582855225, -0.2735871374607086, -2.103036642074585, 1.5546281337738037, -0.45122620463371277, -0.21230176091194153, -0.4135783016681671, -0.24897117912769318, -0.39048510789871216, -0.26372218132019043, 0.2993529140949249, -0.878235936164856, 1.7674057483673096, -1.0682841539382935, -0.3128436803817749, -0.5266236662864685, 0.25184792280197144, -1.428958773612976, 0.10144874453544617, -0.14683791995048523, 0.7619730234146118, -0.6572887301445007, 0.22693629562854767, -0.09667199105024338, -0.8980750441551208, 1.048233985900879, -0.5193896889686584, -0.07952462136745453, 1.0080416202545166, 1.1138635873794556, -0.22392484545707703, -0.5956650376319885, 0.374862939119339, -0.9086552858352661, -0.22998793423175812, -0.28112444281578064, -0.5922819375991821, 0.4113721251487732, -2.2952158451080322, 0.7468686103820801, -0.9418404698371887, -0.18926438689231873, 0.3536957800388336, -0.7875007390975952, -0.8989556431770325, -0.5388107895851135, -1.8111530542373657], [0.4505541920661926, 0.4382113516330719, 2.527709722518921, 1.1038517951965332, -1.742545485496521, 2.0187883377075195, 1.5484119653701782, 0.457584410905838, 0.7397215962409973, 0.6814646124839783, 0.23229935765266418, -0.8015288710594177, -0.726829469203949, 0.12650729715824127, 2.9008195400238037, -1.6926101446151733, 0.7835614085197449, -2.8061106204986572, 0.368316113948822, 0.4061138331890106, 0.9600953459739685, 1.504496693611145, -0.707988977432251, -2.2591452598571777, -0.2634378969669342, 0.46020957827568054, 0.02534884586930275, 2.1491944789886475, 0.5330008268356323, -0.4415243864059448, -0.9191017746925354, 2.4845948219299316, 0.49035701155662537, 2.4896373748779297, -1.133080244064331, 0.3079860210418701, 0.6901519894599915, -0.8035333156585693, 0.7991282343864441, -0.7336450815200806, 0.8037753105163574, -0.4951922297477722, -0.9281389117240906, -0.9337813258171082, -0.6816943287849426, 1.6630703210830688, -0.411151647567749, 0.06600084155797958, 1.3535884618759155, 0.037721239030361176, -1.5599888563156128, -1.614667296409607, -0.23017333447933197, 1.5934255123138428, 1.2818400859832764, 0.16442863643169403, 0.4900038242340088, -0.04814379662275314, -0.6941449642181396, 0.5390342473983765, 0.7136807441711426, 1.8643301725387573, 1.5743170976638794, 0.6554606556892395], [-0.9253189563751221, -0.5682333707809448, 0.9075440764427185, 0.1744469851255417, -0.265778124332428, -0.07850277423858643, -2.8819620609283447, 0.17490048706531525, 2.301297426223755, -1.869564175605774, 0.5188614130020142, -0.22847658395767212, -2.327333688735962, -0.827486515045166, -0.23807936906814575, 0.5855880975723267, 1.0335612297058105, -0.03558114543557167, 0.6437574625015259, 0.5958254337310791, -0.7693244814872742, -0.08488330990076065, -1.1574705839157104, 1.003710389137268, 0.6242976188659668, -0.979286253452301, -0.012911064550280571, 1.6228402853012085, -0.30315151810646057, 0.7347896099090576, 0.986764669418335, -0.37709730863571167, 0.051184963434934616, -1.2876644134521484, 0.6371413469314575, 0.8328503966331482, 0.9489545226097107, 0.540107786655426, 1.9123706817626953, 0.29801127314567566, 1.251676321029663, -2.2982847690582275, 2.489741325378418, 0.9387047290802002, 0.8378251791000366, -0.9857809543609619, 0.4840760827064514, -1.5224919319152832, 1.5638190507888794, 1.3687307834625244, 2.9031336307525635, -1.4240612983703613, 0.0455559566617012, 0.7510946393013, 0.7125610113143921, -2.5060744285583496, 0.5967207551002502, -0.9181232452392578, -0.8994984030723572, 0.5499638319015503, 1.24514901638031, 0.4236088991165161, -1.588800072669983, -0.7045496702194214], [0.8694027662277222, -1.185433030128479, 1.0548758506774902, 1.2767095565795898, 0.12093397974967957, 0.22091247141361237, 0.1288956254720688, 0.7047871947288513, -0.4762002229690552, -0.22721455991268158, 0.03297826647758484, 0.3360816538333893, -2.101231813430786, -0.5833139419555664, -0.2839124798774719, 0.2691114544868469, 0.3714842200279236, -0.3328724801540375, -1.904994249343872, 0.20031942427158356, 0.6640188097953796, 1.0274155139923096, 1.356413722038269, -1.4946670532226562, 0.7715743184089661, -0.7518463730812073, -1.0189284086227417, -0.501229465007782, -0.9145482778549194, -1.4373971223831177, -1.4241811037063599, 1.192487359046936, -0.553073525428772, -0.46575790643692017, -0.11715812981128693, 0.6338682174682617, 1.1876028776168823, -3.085628032684326, -0.8045199513435364, 0.38364824652671814, -0.4765405058860779, -0.10003872215747833, 1.6120412349700928, 1.9910160303115845, -2.7463114261627197, 1.4641063213348389, -0.437783807516098, 0.7919430136680603, 0.5536333322525024, 1.9298378229141235, -0.8338900208473206, 0.10833375155925751, 0.15822605788707733, -1.105347752571106, 0.8937965631484985, -1.3439449071884155, -1.9033399820327759, 0.20996814966201782, 0.7706035375595093, -0.08150014281272888, -1.4972023963928223, 1.0952092409133911, -0.5983183979988098, -0.4903862774372101], [0.5673887133598328, 0.2533649206161499, 1.0469040870666504, 0.3768611550331116, -0.8917185068130493, -0.6633509993553162, -0.5470106601715088, -0.18563252687454224, -0.07622234523296356, -1.299401879310608, -0.11813617497682571, -0.6601577401161194, 0.0007435625302605331, 0.1732393205165863, 0.7592222690582275, -1.0824793577194214, 1.0619336366653442, 0.2425452470779419, -0.07005244493484497, 2.2147738933563232, -0.3411923348903656, -0.587104856967926, -0.8157393932342529, -1.5832065343856812, -0.17371483147144318, 1.1147940158843994, -0.2821740508079529, -0.20896591246128082, -2.113971710205078, 0.15544281899929047, 0.6943492889404297, 0.5986214876174927, 0.5952568650245667, 1.1560217142105103, 1.0638998746871948, -0.4468623101711273, 0.8101563453674316, -0.4229263365268707, 0.8453845381736755, -0.05860712751746178, -0.9627215266227722, -0.265864759683609, 0.9246234893798828, 0.12979577481746674, -0.10757479071617126, -0.2627469003200531, 0.022754762321710587, 1.5836642980575562, -1.113023042678833, -0.1673709899187088, 0.11209968477487564, 1.4671882390975952, 0.24337764084339142, 1.4195612668991089, -0.06379375606775284, 0.2426273375749588, 0.5336084365844727, 0.6951096653938293, 1.4862861633300781, 0.06850089877843857, -0.08883547782897949, 0.09316636621952057, -1.7822952270507812, 1.9495352506637573], [0.22695812582969666, -1.8953889608383179, 0.7733011245727539, 0.050944820046424866, 0.8481139540672302, -0.35539981722831726, -0.5186701416969299, -0.46920403838157654, 0.4795503318309784, -0.6575072407722473, 0.40583106875419617, -0.4745970368385315, 0.5807445645332336, -0.4231571555137634, 0.5007343888282776, -1.0933959484100342, -1.6967341899871826, -0.9105423092842102, 0.7865064144134521, -2.2563583850860596, -0.547990083694458, -0.17067262530326843, 1.291672706604004, -0.6762923002243042, 0.7999534606933594, -0.41303524374961853, 0.21352088451385498, -1.1829124689102173, 0.3598756790161133, 0.03346503898501396, 0.01642499305307865, -0.23006433248519897, 2.5794336795806885, -0.4523774981498718, -0.38629814982414246, -0.7371905446052551, -0.8202660083770752, -0.3524826467037201, 0.2680729329586029, -0.1282341480255127, -0.8279397487640381, 0.5831086039543152, 0.33786216378211975, -0.29411715269088745, 0.591141402721405, -1.8715152740478516, 1.0526506900787354, 0.03841438144445419, 0.23033703863620758, -1.9708402156829834, 1.9357736110687256, 1.8970743417739868, -0.759844958782196, 0.571176290512085, 0.6719393134117126, 1.3853871822357178, 0.39085063338279724, 0.07242800295352936, 1.114660382270813, -0.4078599810600281, 0.9019365310668945, -0.9607275128364563, 0.22234387695789337, -2.100782632827759], [1.5997198820114136, -0.15938039124011993, -0.6235769391059875, -0.04280022159218788, 0.4463523328304291, 0.6279885172843933, -0.855725884437561, -0.8501282334327698, -0.5563551187515259, 0.11805585771799088, -0.548262894153595, 0.7178583145141602, 1.2743626832962036, -0.38122329115867615, -0.42720577120780945, 0.47516191005706787, -0.4433533251285553, -0.2352258563041687, 0.24950745701789856, -1.568787693977356, 0.3499685525894165, 0.3144706189632416, 0.015330623835325241, 1.289176344871521, -0.44787418842315674, 1.458998203277588, -1.1821497678756714, -0.626303493976593, -1.1883058547973633, 0.013026646338403225, 0.9598292112350464, 1.09744131565094, -1.2089364528656006, 0.13463366031646729, 0.3581372797489166, -0.718079149723053, -0.21355734765529633, -0.9283769726753235, 0.5789183378219604, 0.18618443608283997, -0.3020784258842468, 0.447133868932724, -0.48194223642349243, 0.7635406255722046, -0.04709698259830475, 0.9452386498451233, 0.5635396242141724, 0.05509679391980171, 0.3934953212738037, -1.3017833232879639, -0.5581618547439575, -0.6784436702728271, 0.5222094058990479, -1.3718233108520508, -1.5095891952514648, -1.9110543727874756, 1.5281672477722168, 0.23184390366077423, -1.206368088722229, 1.0934884548187256, 0.833605945110321, 1.0197170972824097, 0.8835232853889465, 0.5177732706069946], [1.5401989221572876, 1.8773655891418457, -0.810753345489502, -1.8153680562973022, 1.2485404014587402, 2.27558970451355, 1.8464070558547974, 1.662056803703308, -0.6234716773033142, 1.5731626749038696, -0.48556995391845703, 0.07297848910093307, -1.503095030784607, -0.4393160045146942, -0.665638267993927, 0.2820938527584076, -1.3438231945037842, -0.6149894595146179, -0.3365972936153412, -1.2929633855819702, -0.9255048632621765, -0.6379314661026001, -1.405272126197815, 0.0737396627664566, -1.4362434148788452, -0.4599427878856659, 0.30722618103027344, -0.577938973903656, -1.2651413679122925, 1.2817118167877197, 0.6189768314361572, -0.28456711769104004, 1.2719088792800903, 1.109616756439209, 1.3535833358764648, -0.6508818864822388, -2.747976541519165, -0.2848752737045288, 0.09780729562044144, 0.0634135752916336, 0.36991944909095764, -0.9049534797668457, 0.7756210565567017, 0.5671530961990356, 0.7874922752380371, 0.39465591311454773, 1.0606426000595093, -0.04417213425040245, -0.7287295460700989, 0.6626564860343933, -1.6521632671356201, 0.1325143575668335, -0.2089642733335495, -0.9383512139320374, 0.16000603139400482, -1.6825239658355713, 1.2101192474365234, -0.21989335119724274, 0.03253569081425667, -1.2888990640640259, 0.3998509347438812, 0.3525781035423279, -1.817237138748169, 0.4507679343223572], [0.05339523032307625, 0.6797142028808594, -0.7552192211151123, 2.398925542831421, -0.6216048002243042, 0.29704079031944275, -0.04219400882720947, 1.0331430435180664, -0.7065107822418213, -0.7762122750282288, -0.3883518576622009, -0.9040514230728149, 0.4528103172779083, 0.4486828148365021, -1.0345057249069214, 0.2197599560022354, 0.008872040547430515, 0.13361625373363495, -2.226212978363037, 1.2537611722946167, 0.9825801253318787, 1.8103675842285156, -1.2254698276519775, -2.256150245666504, -0.2815297842025757, 1.2429437637329102, 0.8633999824523926, 0.9958516955375671, 1.0701024532318115, 0.427199125289917, 0.2389741837978363, -0.13531172275543213, -0.16428399085998535, -1.3512942790985107, -0.1078738123178482, -1.4857420921325684, 0.6064075827598572, 1.1099895238876343, -0.4594469964504242, 0.4866607189178467, 0.6238512992858887, -0.2380448281764984, 0.5172916650772095, 1.358764410018921, -0.20426130294799805, 0.6806066036224365, 1.9453179836273193, 0.9606969952583313, -0.6643716096878052, 2.423544406890869, 0.27819693088531494, 0.10166683793067932, 0.5855439305305481, -0.6848886609077454, -0.757465124130249, -1.3335069417953491, -1.1312967538833618, -1.3716254234313965, -0.7117400765419006, 1.7821314334869385, 0.4013826549053192, 0.6297711730003357, -1.3005238771438599, 0.5424007177352905], [0.4602818191051483, 0.6275539398193359, 1.2614988088607788, -0.01450321078300476, -0.6509028673171997, 0.13603760302066803, -1.8729909658432007, -0.0941208004951477, 0.482122004032135, 1.1298415660858154, -1.5224220752716064, -0.2529902756214142, 0.3961573839187622, -1.5078576803207397, 1.2130330801010132, -0.6931873559951782, -0.6600819826126099, -0.5674694180488586, 1.8085649013519287, -0.42601677775382996, 1.1337087154388428, 1.5539075136184692, -0.6971043348312378, 1.2503609657287598, 0.8537035584449768, 0.01786954514682293, -1.1007726192474365, 0.3275447487831116, 0.37140926718711853, 1.0635266304016113, 0.8447513580322266, -0.7578989863395691, 2.468600273132324, -0.0004722443700302392, -0.9100242257118225, -0.37723469734191895, -0.8121662139892578, 1.2367757558822632, 0.6030706763267517, -0.8428544998168945, 0.22450391948223114, -0.8877926468849182, -0.43232885003089905, 0.9087196588516235, -0.623615026473999, 0.6453887224197388, -0.10261429846286774, -1.2955037355422974, -0.4019639492034912, -1.5172256231307983, 1.1656619310379028, 1.6147105693817139, -1.3757907152175903, -0.6500878930091858, -0.7926008105278015, -0.3223164975643158, -1.2147068977355957, 0.3302997052669525, -0.06982506066560745, -0.31143680214881897, 0.36920657753944397, -0.3596149981021881, 0.05615191161632538, 1.4064351320266724], [0.24034668505191803, -0.5157031416893005, 1.001471757888794, 0.207539901137352, 0.16689932346343994, 0.7662056684494019, -0.6907736659049988, 0.18327228724956512, 0.8420333862304688, -0.7472628951072693, 0.06175107881426811, -0.7089453935623169, 1.0618107318878174, 1.9527573585510254, -0.6179239749908447, -0.46693235635757446, 0.25056105852127075, -0.816591203212738, 1.1314057111740112, -0.8011416792869568, -0.45094361901283264, 0.6935063004493713, 1.311049461364746, 1.0833451747894287, -0.12835708260536194, 0.7361353635787964, -1.293858528137207, -0.9887539744377136, 0.49266526103019714, -0.5828373432159424, -0.924553632736206, 0.4056572914123535, 0.6561664938926697, 2.0299205780029297, -0.5082559585571289, -1.1640138626098633, -0.3699983060359955, -0.2721385657787323, -0.1651187539100647, -0.8710676431655884, 0.18784750998020172, 0.2800748646259308, -1.3132861852645874, 2.2480690479278564, 0.24233990907669067, -0.15053433179855347, 1.0799447298049927, -0.19864478707313538, 1.189632534980774, 1.4170547723770142, 0.3061724603176117, -0.7753859758377075, -0.6821789741516113, -0.9253058433532715, 0.3840465247631073, -1.5056958198547363, -2.4500138759613037, 0.007171831093728542, -0.8779549598693848, 1.377184271812439, 0.455013632774353, 1.1462804079055786, 0.32314616441726685, -0.3387082517147064], [0.5870627760887146, -0.4772515892982483, -1.0495405197143555, -0.12069851160049438, -0.8600353002548218, -0.22068847715854645, 1.3919289112091064, 0.30505886673927307, -0.1502419114112854, -0.9597964286804199, -0.7493109703063965, 0.6738193035125732, -1.1185129880905151, 0.0970914363861084, 0.8257302045822144, 0.4515041708946228, 0.2053101807832718, 0.8996176719665527, 1.3500616550445557, 0.6566942930221558, 1.3448829650878906, 1.25149405002594, -0.6120417714118958, -1.9035500288009644, 0.24875669181346893, -0.6473737359046936, 0.1458103209733963, -0.300810307264328, -0.11798503249883652, 0.04756544902920723, -0.6274322271347046, 0.48770108819007874, 0.7500859498977661, 0.45964643359184265, 0.4524545669555664, -0.14232806861400604, -0.5129624605178833, 0.10207788646221161, -1.037623405456543, 2.166827440261841, -0.3888963758945465, 0.5447772741317749, 1.9239689111709595, -1.6829991340637207, -0.23905587196350098, 1.015141487121582, 0.12528470158576965, -1.2394639253616333, 0.50547194480896, 1.6508138179779053, 1.3458162546157837, -0.1374945491552353, -0.19384540617465973, 1.5872094631195068, 0.21565939486026764, -0.37267303466796875, -1.6337542533874512, 0.2934936285018921, -1.9539084434509277, -0.6309617757797241, 1.557336449623108, 0.5000385046005249, -1.2881900072097778, 1.3007776737213135], [0.22443652153015137, -0.7693566679954529, -0.736158549785614, 1.9415098428726196, 0.3790823817253113, 0.8946153521537781, -1.300636649131775, -0.8488706946372986, 0.0495857335627079, 1.1026068925857544, -0.05619501695036888, 0.4013594388961792, -0.9451006650924683, -0.8803812265396118, 0.6254278421401978, 0.25641730427742004, 1.2321581840515137, 1.2863129377365112, -0.8932860493659973, -0.18021787703037262, -0.6375786662101746, -0.7762031555175781, 0.3746989667415619, 0.24276648461818695, 0.2291543334722519, -0.5785586833953857, -0.6204627156257629, -1.0940487384796143, -0.33713677525520325, 0.645349383354187, 0.33310866355895996, -1.5806808471679688, -1.1316713094711304, -0.7155872583389282, -1.2105708122253418, 0.17685121297836304, 0.2006312608718872, 0.5777363181114197, 1.409264326095581, -1.4205174446105957, -1.293888807296753, -0.91986083984375, -0.6592749953269958, -0.9068781137466431, 2.6743886470794678, -1.72947359085083, -0.5950736403465271, -0.5755519866943359, 0.10132351517677307, -0.759455680847168, 0.17176079750061035, 0.8411312699317932, -0.023362796753644943, 1.296105146408081, 0.4405336081981659, 0.34805384278297424, 0.5241764783859253, -0.3325697183609009, -1.9256151914596558, -0.08792151510715485, -0.37945985794067383, -0.2180025577545166, 1.0202215909957886, 0.8084489107131958], [-0.5233486890792847, -1.882733702659607, -1.3653912544250488, 0.5258573889732361, -0.9887242913246155, -0.8079501390457153, 0.7832930088043213, -1.8635739088058472, 1.8999981880187988, 0.01644888147711754, 0.8923677802085876, 0.7925323843955994, -1.3511830568313599, 1.8059272766113281, -0.4882534146308899, 0.2368078976869583, 1.269917607307434, -1.1148266792297363, 0.8975223898887634, -0.5967954993247986, 0.32340583205223083, 0.3037431538105011, -1.334682583808899, 0.6639023423194885, -0.0661124587059021, 0.12689514458179474, -0.1657162457704544, -1.5518044233322144, 0.35814109444618225, 0.45996397733688354, 0.4485775828361511, 1.535270094871521, -0.631813108921051, -0.3615489900112152, 0.0016209784662351012, -0.21853578090667725, -0.08670289814472198, -0.3261369466781616, 0.2730446755886078, -0.9641191363334656, -1.0758730173110962, -1.5052350759506226, 1.5204495191574097, -0.577829897403717, -1.562817096710205, 1.0648809671401978, 0.05389877036213875, 1.5940711498260498, 0.37897834181785583, 0.16161423921585083, 0.4680723547935486, 0.8995766639709473, 0.6359112858772278, 1.0976314544677734, 0.8773963451385498, 0.831208348274231, 0.6224852204322815, 0.3745019733905792, -0.09113072603940964, 1.0798168182373047, 0.6084833741188049, 1.8840597867965698, -1.3806300163269043, 0.979194700717926], [-0.7589668035507202, -0.8793250918388367, 2.070894956588745, 0.7610850930213928, 1.390377402305603, -0.0020849707070738077, -1.4694291353225708, 1.0441175699234009, 1.378551721572876, 0.42780858278274536, -0.8047189116477966, -1.1948622465133667, 0.20866747200489044, 0.32570332288742065, -0.7001480460166931, 0.7431901097297668, -0.9363253712654114, -0.433657169342041, -1.3343762159347534, -0.8093536496162415, -0.606474757194519, -1.4261044263839722, -0.8306807279586792, -0.9058226943016052, -1.3777168989181519, 1.2496174573898315, -0.4618164598941803, 0.2764992117881775, 0.712008535861969, 0.03449025750160217, -1.26058030128479, -0.24118897318840027, 0.04804004728794098, -0.9057800769805908, -1.2639360427856445, 1.470133900642395, -1.3301522731781006, -1.064695954322815, -1.561265468597412, 1.263890266418457, -0.6158575415611267, -0.6003271341323853, -0.6681762337684631, -0.6589166522026062, -0.7284427285194397, -1.0664255619049072, 0.9727939963340759, 1.0810474157333374, -0.4291798770427704, -1.9144083261489868, 0.18770411610603333, 0.4246246814727783, -0.0982702299952507, -0.2704564332962036, -1.977786660194397, 0.8631482720375061, 2.0194027423858643, 0.1360052078962326, -2.4104714393615723, 1.0252200365066528, 1.2788825035095215, 0.5674927830696106, -1.6232191324234009, 1.6251907348632812], [0.5919304490089417, 0.4021835923194885, 0.9744415879249573, 0.4310917258262634, 0.1375226080417633, -0.6026577949523926, 0.6289094090461731, 0.4091668725013733, 0.2691943049430847, 0.5475264191627502, 0.8124317526817322, -0.08557651937007904, -0.6567620038986206, 2.1786398887634277, -1.4287254810333252, -0.18344514071941376, -0.8111551403999329, -0.9587236046791077, -1.386872410774231, 0.08561813831329346, 1.5359922647476196, 0.027815574780106544, -0.4297384023666382, 1.8822271823883057, -0.22671401500701904, 0.10922808200120926, 0.7943025827407837, -1.0694069862365723, 1.2716717720031738, -1.8465547561645508, 0.21063324809074402, -0.41363638639450073, -1.2650249004364014, 0.06803632527589798, 1.1212750673294067, -1.0543887615203857, -0.007999733090400696, -0.3944253623485565, 0.3563993275165558, -0.9709354043006897, -0.40955469012260437, 2.0407955646514893, 0.35069623589515686, 1.2790894508361816, -0.7150378227233887, 3.226160764694214, 0.9270268678665161, -0.6604732275009155, -0.9089975357055664, 0.20163138210773468, -0.48503217101097107, -0.45870155096054077, 1.1114320755004883, 2.3304502964019775, -1.5484191179275513, -1.669415831565857, 1.310983657836914, -0.5300897359848022, -0.4479527175426483, 0.13476425409317017, 0.16400322318077087, 1.073826551437378, -0.2350839376449585, 0.13803574442863464], [-0.35283541679382324, -3.56612491607666, -0.8732483386993408, 0.3682714104652405, 0.20016710460186005, -0.013478772714734077, 0.3683432936668396, -1.1443580389022827, -0.23512202501296997, -0.0006736793438903987, 1.303694248199463, 0.16882559657096863, -0.7279202342033386, 0.1385665088891983, 0.8425002098083496, -0.7445522546768188, 0.1371680051088333, -1.22557532787323, -0.015656666830182076, -0.6306298971176147, -0.12566974759101868, -2.1350302696228027, 1.8484693765640259, -1.509586215019226, 0.19004912674427032, 0.8317515850067139, -0.16157618165016174, 1.3073320388793945, 1.6219881772994995, 0.5612233281135559, -0.12148027867078781, -1.5369641780853271, 1.2885468006134033, 0.48382115364074707, 0.08018175512552261, -0.3306192457675934, -0.4220910668373108, -0.8707770109176636, 1.286827564239502, -0.728886604309082, 1.3908995389938354, 0.10402249544858932, 0.510472297668457, -0.24000345170497894, -0.9532433748245239, -0.2683306336402893, 0.8728113770484924, -3.736276149749756, 0.7327476739883423, 1.40285062789917, -0.7766905426979065, 0.287876695394516, 0.5373445153236389, 0.9602118730545044, -1.2715373039245605, 1.1824809312820435, -0.18338508903980255, 1.777495265007019, 1.3921723365783691, -0.5314928293228149, 1.3976384401321411, -1.1820428371429443, 0.3527818024158478, -0.3607339859008789], [-0.6305368542671204, 0.7099171280860901, -0.073741614818573, 0.14682160317897797, 1.2751713991165161, -0.716701328754425, -0.5262661576271057, 0.685063362121582, -0.5157088041305542, -0.5640579462051392, -1.1062672138214111, -0.06383679062128067, 0.7298159003257751, 1.0513051748275757, 0.6928272247314453, 0.15724973380565643, 0.9518593549728394, -0.4740053713321686, 1.1181060075759888, 0.007448362652212381, 1.7325263023376465, 0.08556555956602097, 1.769139051437378, -1.4757323265075684, -0.24475617706775665, 0.8752462863922119, -0.45730847120285034, -0.17535611987113953, -0.9513586759567261, 0.44411197304725647, -0.3651186227798462, -0.5474072694778442, 0.45138019323349, -0.06901956349611282, 1.11165189743042, -1.0790024995803833, -0.17946375906467438, -0.6095919013023376, 1.2856178283691406, -0.6476185321807861, 0.8721048831939697, -0.5301474332809448, 0.3608664870262146, 0.3811734914779663, 1.2306994199752808, 1.2940939664840698, 0.09362620115280151, -0.23508578538894653, -0.7388066053390503, -1.2323938608169556, 1.1685580015182495, 1.0963181257247925, -0.7330822348594666, 1.0127098560333252, 0.09357914328575134, -0.6120973229408264, -0.917192816734314, -1.1318663358688354, 0.6099302172660828, 1.314297080039978, -0.7690548300743103, 0.8558558821678162, 1.5811599493026733, 1.1880643367767334], [1.5346194505691528, -0.09321258217096329, -0.35428133606910706, 2.1393744945526123, -0.8487149477005005, -0.25462064146995544, -0.7003440260887146, 1.1229770183563232, -0.7642678022384644, -1.3644906282424927, 1.0707664489746094, -0.2072582095861435, 0.48584452271461487, 0.8232678771018982, -0.3540189862251282, -0.0882442444562912, 0.7010863423347473, 2.3516716957092285, -1.5532481670379639, 0.37294721603393555, 1.0054662227630615, 0.5410547256469727, 0.7478616833686829, 0.7052071690559387, 0.6465593576431274, 2.180370330810547, -2.255190372467041, -0.48339346051216125, 1.6396187543869019, -0.06816421449184418, -0.18957553803920746, 1.4621891975402832, 0.3308534622192383, -1.607168436050415, 0.9191515445709229, -1.287214994430542, 0.35663753747940063, -1.329053282737732, -0.2956671118736267, 0.20534349977970123, -1.221542239189148, 0.5171700119972229, 1.3468316793441772, -1.5627422332763672, 0.3349273204803467, -0.9355971813201904, 1.0617718696594238, -0.49187594652175903, 0.19214637577533722, 1.9347681999206543, 1.4148244857788086, -0.39025232195854187, -0.13587835431098938, -0.859832763671875, 0.6642683744430542, 0.5570008158683777, 1.5441954135894775, -0.8386828303337097, -0.2259664535522461, -0.4175649881362915, -1.205297589302063, 1.5111013650894165, 0.021144308149814606, 1.3340494632720947], [1.3894619941711426, 0.8367565274238586, -0.4345906972885132, -1.2929387092590332, -1.5666239261627197, 0.2739506661891937, -0.2740960121154785, -2.174553871154785, -0.561418890953064, 0.28126272559165955, 0.21208330988883972, -0.6870006322860718, -0.1522296816110611, 0.25438979268074036, -0.3907463848590851, -1.4821298122406006, 0.4585730731487274, -0.09290134161710739, 1.8314906358718872, -0.24137195944786072, 1.5865808725357056, 1.8923561573028564, 0.1324191391468048, 1.4161746501922607, -1.0293651819229126, -0.2551071345806122, 2.266188621520996, 0.8629880547523499, -1.4634032249450684, -0.2347993701696396, 0.01853647269308567, 0.844760537147522, 0.016153130680322647, -1.0867887735366821, -0.04593414440751076, 0.5595819354057312, -0.0882074236869812, 0.8042819499969482, 2.376115322113037, -0.4172517955303192, 0.6412023901939392, -1.5625544786453247, 0.38550081849098206, 0.15459279716014862, -0.5806333422660828, -0.1892162561416626, -0.30836978554725647, -2.253920555114746, 2.291344165802002, -0.40581515431404114, 0.05929531529545784, 1.5747917890548706, 1.0531419515609741, 0.45452919602394104, 0.4734468460083008, -1.7518235445022583, 0.08165138214826584, 0.9832277894020081, -1.3069809675216675, -0.215581014752388, 1.4018771648406982, -0.18981096148490906, -1.7858110666275024, -0.13471558690071106], [0.8329178094863892, -0.14114217460155487, 0.8266572952270508, 0.26152995228767395, -0.7652356028556824, 1.2541536092758179, 0.6448298692703247, -1.6971811056137085, -0.2904592454433441, 1.2127041816711426, -0.5044068098068237, -0.693310558795929, -0.021470556035637856, 0.5024685859680176, 1.0065349340438843, 0.23588475584983826, -0.4545474052429199, 0.5386192202568054, 0.3832935094833374, -1.233122706413269, 0.09709498286247253, -0.774163007736206, -0.14609619975090027, 1.6628389358520508, -0.49390971660614014, 0.061270616948604584, 0.9248623847961426, -0.6060705780982971, 1.975498914718628, 0.8823691010475159, -0.6257011294364929, -0.660553514957428, -0.5638076663017273, 0.9980373978614807, 1.5857635736465454, 1.070655107498169, -0.5031610131263733, -0.8635998368263245, 0.21861976385116577, -0.08511550724506378, 0.12580421566963196, 0.7511044144630432, -0.17625774443149567, -0.5678699016571045, -1.1332223415374756, 1.8607685565948486, -0.4598943591117859, -0.08618233352899551, 0.17575156688690186, 1.1594715118408203, 1.5129988193511963, 0.2951885461807251, 1.5463602542877197, -0.8633037805557251, -0.18338970839977264, 0.4708685874938965, 1.4145866632461548, 0.3701929450035095, -0.7357673645019531, 2.1972298622131348, 1.5251468420028687, 0.47966495156288147, -0.3301825523376465, -2.044912815093994], [-0.30022570490837097, -0.5339559316635132, -0.08836860954761505, -0.2184954732656479, -1.3524526357650757, -0.8315297365188599, 1.7652018070220947, -1.0798213481903076, -1.2221753597259521, 1.2145037651062012, 0.9997789859771729, -0.58149254322052, -1.8806360960006714, -0.45984795689582825, -0.485660582780838, 0.5182586312294006, -0.32207077741622925, 0.4056597352027893, 0.2883857488632202, 1.0706020593643188, 1.0805600881576538, 0.32047954201698303, -0.5581847429275513, 0.09926073998212814, -0.47878819704055786, -0.5209017992019653, 0.21988199651241302, 1.1071422100067139, -1.457381248474121, -1.2522023916244507, 0.4612077474594116, 0.8249226212501526, 0.7946102619171143, -0.35958045721054077, 0.8639816045761108, -0.3629107177257538, -0.918728232383728, -0.8484472632408142, -0.3354244828224182, -0.0794253870844841, -0.06395664066076279, -0.3047609329223633, -1.0123625993728638, 3.1857757568359375, -0.891235888004303, -1.0338529348373413, 2.0106217861175537, -1.8830044269561768, 0.6851017475128174, 0.2587817907333374, -0.7937166690826416, -2.103041887283325, -0.9833704829216003, 2.127253532409668, 0.8929941058158875, -0.9420478343963623, -0.12545952200889587, -1.12873375415802, 0.35318055748939514, -0.027671849355101585, -0.37004438042640686, 1.1078788042068481, 1.2760969400405884, -0.2368801087141037], [0.24748636782169342, 0.14762601256370544, -0.4908321797847748, -0.21923664212226868, -0.6312587857246399, 1.3755125999450684, -1.125679850578308, -0.7832589745521545, 0.22095482051372528, 0.3141593933105469, 0.6912404894828796, 0.31054815649986267, -0.2747732400894165, -0.05905454233288765, -1.4884095191955566, 0.4360193610191345, -1.142108678817749, -1.2154130935668945, 0.05686943605542183, -0.05225317180156708, 0.10700792074203491, -0.10565290600061417, 0.305646687746048, -0.6113876700401306, -1.6580382585525513, -0.4699883759021759, -0.08169315755367279, -1.00441575050354, 1.2890563011169434, -2.2018883228302, 0.33422350883483887, -1.0474828481674194, 2.7711246013641357, 0.7333793044090271, -0.23893392086029053, -1.4719525575637817, -0.30357372760772705, 0.14946706593036652, -0.36617225408554077, 1.9938173294067383, -0.6919885277748108, -1.046342134475708, 0.5863966345787048, -0.5103510022163391, 0.4688419699668884, -0.1017310619354248, 0.8885071277618408, 0.41626694798469543, 1.4262895584106445, 0.7616738677024841, 1.5854741334915161, 1.3641295433044434, -0.644534170627594, 0.6146693229675293, 0.7461830377578735, -0.3871289789676666, -1.3370599746704102, 0.28865769505500793, 1.0248701572418213, 0.36377063393592834, -2.692023992538452, -0.36850085854530334, 0.18289753794670105, 1.1327331066131592], [-0.7634231448173523, -0.48345088958740234, -0.5055869817733765, -0.9664134979248047, 0.20378544926643372, 0.18328113853931427, 0.31963223218917847, -0.3991391062736511, -0.5735726356506348, 1.3948907852172852, -1.9775599241256714, -0.3528631031513214, 0.7423486709594727, 1.2558610439300537, -1.1092822551727295, 0.18562772870063782, 2.2397422790527344, -0.33143511414527893, -0.6412315964698792, -1.1752148866653442, 0.08318065851926804, -1.9695810079574585, 0.7437012195587158, -1.2067112922668457, 0.5717282891273499, -1.6119393110275269, 1.4250051975250244, -1.1914440393447876, -0.8374466896057129, 0.5095211267471313, 0.12096893787384033, -0.26305457949638367, -2.0124852657318115, -0.033840373158454895, 0.45280003547668457, 1.2533806562423706, -0.2502448260784149, 0.6085770726203918, -1.0566778182983398, -0.35432013869285583, -0.2720795273780823, 0.6479017734527588, 0.2558324933052063, 1.562137246131897, -1.430282711982727, -0.5848899483680725, -0.9199479222297668, 0.9399317502975464, 0.6258029937744141, 0.045755524188280106, -0.3888700008392334, -1.4437029361724854, -0.39442962408065796, -2.0461480617523193, -0.6165794730186462, 0.25549954175949097, -0.289010614156723, -0.6472007036209106, 0.9436049461364746, 0.6718719601631165, -0.09779246896505356, -0.07949409633874893, 0.5310858488082886, -0.6146133542060852], [-0.3203561007976532, 0.5547333359718323, -0.8615719079971313, -0.29685837030410767, -0.30609118938446045, -1.435647964477539, -0.8571922183036804, -0.18710696697235107, 0.4775449335575104, 1.075230360031128, -1.8400641679763794, 1.0584794282913208, 0.35296010971069336, -2.2783315181732178, 1.259741187095642, 1.3203651905059814, 0.5576949119567871, -0.22305704653263092, 0.36725014448165894, -0.03713414818048477, -0.60121089220047, 0.3653298020362854, -0.9941428303718567, 0.70576411485672, 0.26564398407936096, -2.037517547607422, 1.4097058773040771, 1.1422524452209473, -2.039334297180176, 0.510167121887207, 0.41265323758125305, 0.39049196243286133, -1.2307072877883911, -0.3870300352573395, -0.38415855169296265, -0.2687872648239136, -0.3645740747451782, -1.2160239219665527, -0.4989245533943176, -0.1870521605014801, 0.46749335527420044, 0.4844357371330261, -0.21681621670722961, -0.014091316610574722, -0.07651934027671814, -1.008127212524414, -0.04536675289273262, 1.0239914655685425, -1.1948113441467285, 0.10032138973474503, -1.6140422821044922, -0.481413871049881, -1.0779101848602295, 1.5397037267684937, 0.5151314735412598, -0.7836657166481018, 0.7624372243881226, 0.06173355132341385, 0.1504705548286438, -0.04485488310456276, -0.7034075260162354, 0.229305699467659, -0.26150742173194885, -0.03085101954638958], [-0.18839126825332642, 0.4372832477092743, -0.5360826849937439, -0.3237132132053375, -1.9992059469223022, -0.6179156303405762, 0.9449945688247681, 1.1174595355987549, -1.2738730907440186, 1.4488599300384521, -1.627110481262207, -0.028997279703617096, 1.3591294288635254, 1.1004844903945923, -0.28089359402656555, 0.7857255339622498, 0.3479682207107544, 0.646862268447876, 1.018703818321228, 0.8306294679641724, -1.778302550315857, 0.5838227272033691, -1.3255482912063599, 0.5317502021789551, 0.6866759657859802, 0.6814835667610168, 0.34693965315818787, -0.8598402738571167, 1.0531980991363525, 0.031756430864334106, 1.4875142574310303, 1.2442599534988403, 0.022664429619908333, -0.5943516492843628, 1.3382378816604614, 1.458044409751892, 0.5289067625999451, -1.6144752502441406, -2.085616111755371, -0.552330732345581, 0.30818113684654236, -0.8004938960075378, 0.21919739246368408, 0.23951581120491028, -0.267270565032959, -0.8982307314872742, -0.27819496393203735, 0.02882496453821659, 0.18926088511943817, 1.1239513158798218, -0.621288537979126, 1.199431300163269, -1.82234787940979, -1.0671627521514893, 0.4831239879131317, 0.12224284559488297, 0.3076150715351105, -0.38825348019599915, 1.1526010036468506, -0.30410856008529663, 0.7453442811965942, 0.7291285991668701, 1.141493797302246, -1.3030203580856323], [0.11588738858699799, -0.40620478987693787, -1.395982265472412, 0.9519129991531372, 0.6333864331245422, 2.230295181274414, 0.9204311370849609, -0.34824687242507935, -0.026859186589717865, 0.06980778276920319, -0.2836189270019531, -1.1367725133895874, 2.1538174152374268, 0.1997620016336441, 0.8983786702156067, -0.7092254161834717, -2.1736578941345215, -0.19360701739788055, -0.41006627678871155, -0.27564355731010437, 0.7984306216239929, -0.6715317368507385, 1.7218607664108276, 0.22057227790355682, -1.4154223203659058, 1.9615150690078735, -0.7609426975250244, -0.5185715556144714, 0.12576943635940552, 1.0710946321487427, -0.3216639757156372, 0.32290399074554443, 1.860364317893982, -0.7743463516235352, 0.3832607865333557, -0.07218381762504578, -1.522344708442688, -0.5697990655899048, 0.3344692289829254, -1.5273257493972778, -1.411417007446289, -0.4557524621486664, 0.19267600774765015, 0.37880387902259827, 0.5422149896621704, -0.5809873938560486, 0.9600743651390076, 0.8258916139602661, -1.1725883483886719, -1.384550929069519, -0.4158477187156677, 0.9539659023284912, 1.1203492879867554, -2.0090198516845703, 0.39270174503326416, 1.8255356550216675, -0.966428279876709, -0.30295106768608093, -0.7654547095298767, 0.5000942945480347, -1.326767921447754, 0.9862050414085388, 0.46608883142471313, 0.018012456595897675], [-0.41512852907180786, 0.044882725924253464, -0.8093365430831909, 0.3830414414405823, -1.0168076753616333, 0.46159955859184265, 0.1773238182067871, -0.6286220550537109, 0.11087355762720108, -0.6319384574890137, 1.671325922012329, -0.09421318024396896, 0.640670895576477, -0.30249494314193726, 0.0659131109714508, -1.4517849683761597, 0.29878565669059753, -0.24422863125801086, -1.356139063835144, 1.5402750968933105, -0.7572070360183716, -0.60544353723526, -0.905209481716156, 0.7976596355438232, 0.211816668510437, -0.3327862620353699, -0.43946462869644165, 1.4621241092681885, -1.9156930446624756, -1.5028085708618164, -0.40998727083206177, 1.264288067817688, 1.4280608892440796, 0.2585037648677826, -1.6243364810943604, -1.3394228219985962, 0.21066436171531677, -0.07994990795850754, -0.6890914440155029, -0.6929187774658203, -0.17853578925132751, -0.3180927038192749, -1.6152466535568237, 1.2226163148880005, 2.5009424686431885, -0.952803909778595, 0.695496678352356, -0.644659161567688, -0.9939099550247192, -1.9438189268112183, 0.9056767821311951, -1.4709813594818115, -0.10942193865776062, 1.2906094789505005, -1.2115585803985596, 0.539230465888977, -1.7476314306259155, 1.0221623182296753, 0.04480687156319618, -1.005091905593872, 1.3700594902038574, -0.20523017644882202, -0.3443024754524231, 2.503446340560913], [0.9109674692153931, 0.7156429290771484, -1.649124026298523, -0.5120260715484619, 0.6819326877593994, 0.13512806594371796, 0.6000871658325195, -1.2961691617965698, -0.7505214214324951, -0.6644171476364136, -1.3856122493743896, -0.1277453899383545, -0.04522179439663887, -0.6100857853889465, -0.926478385925293, 0.25147223472595215, 0.5882575511932373, -0.2556556761264801, 0.9012711048126221, 0.2291906476020813, -0.9179860353469849, -0.2117285281419754, -0.3749485909938812, -0.41597455739974976, 0.9228962659835815, -1.1579781770706177, -0.04012959450483322, 0.6892421841621399, -0.8695840239524841, -0.05190083757042885, -0.590899646282196, 0.7440297603607178, 0.006788293365389109, 1.383609652519226, 0.9066503047943115, -0.715816855430603, -1.0089191198349, 1.051114559173584, 1.401023030281067, 0.8631550669670105, 0.17268487811088562, 0.15384413301944733, -1.5828416347503662, 1.3159708976745605, -0.3753228485584259, -0.5190495848655701, 0.013080253265798092, -0.03437066823244095, 0.8289609551429749, -0.45015594363212585, -0.9718217849731445, 0.47877776622772217, 0.15401791036128998, -1.980773687362671, -1.959304690361023, -1.489081621170044, -0.3122512102127075, 0.6966802477836609, 1.815028190612793, 0.7599107027053833, 0.3794618248939514, -0.09786448627710342, -0.26476603746414185, -0.8235512971878052], [-0.3419411778450012, -0.19389764964580536, -0.8720552325248718, 2.3792450428009033, -2.2753188610076904, 0.8315421342849731, -2.292865037918091, -0.5503708720207214, -0.026340974494814873, 0.38047540187835693, -0.09991078078746796, 0.5458962321281433, 0.8519248962402344, -0.9717555642127991, 0.5859193205833435, 2.210573434829712, -0.8694413304328918, 0.9797579646110535, 1.4695988893508911, -0.4596472382545471, -0.32153499126434326, -1.5617631673812866, -0.06705674529075623, 0.37220650911331177, -0.13384172320365906, 1.0431140661239624, 0.6107084155082703, -0.028245653957128525, -0.08145830035209656, 0.02205173298716545, -2.9386777877807617, 1.1608047485351562, -1.1475520133972168, 2.1231226921081543, 1.540183186531067, -1.0735152959823608, 0.44892314076423645, -1.4711066484451294, -0.37183788418769836, -0.40859290957450867, 1.0279042720794678, 0.32075971364974976, 0.07891745865345001, 2.2666308879852295, 0.4242140054702759, -0.8189648389816284, 0.6520425081253052, -1.9813246726989746, 1.0465208292007446, 0.7953764796257019, -0.2442263662815094, 1.3735953569412231, -1.044655442237854, 1.1800388097763062, 0.1637711524963379, -0.24182921648025513, 1.326422929763794, -0.836395800113678, 1.9676408767700195, 0.25921621918678284, 0.2154267132282257, 0.8578128218650818, -2.467238187789917, -0.4999397099018097], [-0.7999408841133118, 0.33455273509025574, -0.9352160692214966, -1.0708390474319458, -0.18499250710010529, -0.9277823567390442, -0.9023212790489197, -0.2601906359195709, -0.10212007910013199, -1.7200260162353516, 1.0584967136383057, 0.6690154671669006, 0.8620299696922302, 1.108084797859192, 0.6165526509284973, 0.8835522532463074, 1.4093239307403564, -0.6076843738555908, 0.7532927989959717, -0.44862017035484314, -0.8281657099723816, 0.36285728216171265, -0.3106668293476105, 0.38052433729171753, -0.8036489486694336, 0.8430378437042236, 0.8645458817481995, 1.506181240081787, -0.3101005256175995, -0.5695826411247253, 2.1030192375183105, 1.7837839126586914, -1.7590798139572144, 0.738596498966217, 0.8529759645462036, -1.5667989253997803, 0.01730257086455822, 0.22205939888954163, -0.9868194460868835, -0.3208213448524475, -2.021946430206299, -0.32969120144844055, -0.907170832157135, 0.12436053156852722, 1.228410243988037, -0.48148801922798157, -0.6493218541145325, 0.6490858197212219, 0.2426467090845108, 0.9460350871086121, -0.6042481660842896, 0.6296421885490417, 0.6957489252090454, -0.4528293013572693, 0.877366304397583, 0.1255805939435959, -0.2655169367790222, -0.924262523651123, -0.10233421623706818, 1.7875711917877197, 0.5497410893440247, -0.7062466144561768, 0.340200811624527, 0.48761457204818726], [-0.8607261180877686, -1.3067295551300049, 1.486710786819458, -1.0744043588638306, -0.3217794597148895, 1.1220693588256836, 1.4745099544525146, -0.3876083791255951, 1.3345381021499634, 1.3042669296264648, 0.36888405680656433, -1.1159089803695679, -0.10585306584835052, -0.8233312964439392, -1.724806547164917, -0.895631730556488, 0.3654994070529938, -0.6071300506591797, 1.1793493032455444, 1.0536549091339111, -1.133554458618164, 1.6196190118789673, -1.2959262132644653, -0.15528742969036102, 0.8969399929046631, 0.4820622503757477, 2.4649994373321533, -1.990376353263855, -0.3668968975543976, 1.2217018604278564, -0.5114555358886719, -0.7168154716491699, 0.6314718723297119, -1.447446584701538, -0.5548676252365112, -2.046199083328247, 0.3534209728240967, -1.3884334564208984, -0.8702161312103271, 0.9947215914726257, -0.2671107053756714, -0.42841875553131104, 0.6552450656890869, 0.3523177206516266, -1.7855064868927002, 0.6832349300384521, 0.9460785984992981, -0.06321068853139877, 0.665666401386261, 0.874708354473114, 0.3525877594947815, -2.0445423126220703, 0.9480639696121216, -0.27628588676452637, -1.5664427280426025, 1.5706077814102173, -0.8274861574172974, -0.23101919889450073, -1.9365519285202026, 0.6432188749313354, 1.3509632349014282, -1.1988710165023804, -0.6170763373374939, -0.14683406054973602], [0.07667554169893265, 1.0743374824523926, -1.9061646461486816, 0.6059749722480774, -1.6800175905227661, 0.8238641023635864, 0.03706367686390877, -1.1139333248138428, -0.23002320528030396, -0.7783544063568115, -1.1932940483093262, -0.5144147872924805, -0.4316098093986511, -0.08491970598697662, 0.25043919682502747, -0.49395111203193665, -1.020077109336853, -0.5193319320678711, 1.175008773803711, 0.5506091713905334, 1.7284873723983765, 2.2722976207733154, 2.0666303634643555, 2.1122097969055176, -0.24859067797660828, -1.2789356708526611, 1.5111373662948608, -0.7567680478096008, -1.5462753772735596, -0.5068456530570984, -1.3936611413955688, 0.13368551433086395, 0.21281763911247253, -0.235316202044487, -1.3920069932937622, 0.6428788304328918, 0.7846809029579163, -0.43888553977012634, 1.5924632549285889, 0.10304680466651917, -0.18402519822120667, -0.3574936091899872, -1.1280419826507568, -0.10461907833814621, 0.49280819296836853, -1.6636734008789062, 1.4157577753067017, -0.08507001399993896, 1.932605266571045, 0.354449063539505, -1.223280668258667, 0.2342846393585205, -0.01310538500547409, 0.8252639174461365, -0.5922023057937622, -0.7517277598381042, -0.4346718192100525, 0.36655306816101074, -1.3765641450881958, -0.13787299394607544, -1.1091530323028564, -1.789336085319519, -0.2287394255399704, 2.579007625579834], [1.1067758798599243, -1.3070778846740723, 0.5697370171546936, 1.4563946723937988, 1.6880912780761719, 0.2660076916217804, -0.8804709911346436, 0.20013658702373505, 0.02275701053440571, -1.8077600002288818, -1.1182515621185303, -1.3222765922546387, 0.8354485034942627, 0.7538686990737915, 0.07049576193094254, -1.630230188369751, 0.1192309707403183, -2.0835862159729004, 1.6924525499343872, -0.2298271209001541, -0.9683383703231812, -1.85100519657135, 0.056498363614082336, -0.17821018397808075, 0.13472303748130798, -0.23375636339187622, -0.33821743726730347, 1.0932549238204956, 0.5200738906860352, -0.387846440076828, 2.06284236907959, 0.2859480381011963, -0.13088946044445038, -0.565441906452179, -0.9459766745567322, 0.06341048330068588, 0.1937403529882431, -0.8683448433876038, -0.8178548216819763, -2.3014938831329346, -1.6412298679351807, 1.0117952823638916, -1.4666498899459839, -0.5997819304466248, 0.780156135559082, -0.2686845660209656, 1.0592772960662842, 2.335547685623169, -0.19058240950107574, -1.3176862001419067, -3.1887009143829346, -1.4554316997528076, -1.3897547721862793, -0.0587431974709034, -0.9145363569259644, 1.0564199686050415, -1.480066180229187, 0.18916548788547516, -1.4319957494735718, 0.011790212243795395, 0.6303379535675049, 1.3026777505874634, -1.1525866985321045, 0.5054472088813782], [0.40514490008354187, -1.8159221410751343, -0.7296311855316162, -1.2962132692337036, 0.24371719360351562, -0.2885705828666687, 0.23343035578727722, -0.4641992747783661, -1.7552756071090698, 1.538678765296936, -0.4696648418903351, 0.5208256244659424, 0.11253675818443298, -1.018211841583252, -0.1971164345741272, 1.7917158603668213, -0.7480925917625427, -1.8830673694610596, 0.5291978716850281, -1.2426111698150635, 1.5967438220977783, -0.529007077217102, 0.48124414682388306, 0.44836950302124023, -1.045868992805481, 0.8656987547874451, 0.13524088263511658, -0.380434513092041, -2.120474100112915, 1.116668939590454, -0.8037371635437012, -2.785655975341797, -0.1512320637702942, -0.2731793224811554, -0.34130117297172546, -0.5465530753135681, 2.1104161739349365, -0.2822323143482208, -0.2978508770465851, 0.16827814280986786, -0.1734437644481659, -0.12334272265434265, 0.10638002306222916, 0.5884759426116943, 0.2045133113861084, -0.5725557208061218, -1.1479288339614868, 0.6317551732063293, 0.5373760461807251, -0.9886460304260254, -1.050901174545288, 0.6452428698539734, 0.8041101694107056, -1.2735556364059448, -0.6138441562652588, 0.3246679902076721, 1.897007703781128, -0.3679327070713043, -0.6880382299423218, -0.389633446931839, -0.7104266881942749, 0.6105976104736328, 0.3536908030509949, 0.2490256130695343], [0.5613445043563843, 0.7007573246955872, 0.1352514773607254, -1.1206278800964355, -1.9397138357162476, 0.1208401694893837, 0.08065246790647507, 0.22909487783908844, -0.8304994702339172, -0.4558204412460327, -0.8063400983810425, -0.014687268063426018, -0.22754085063934326, 1.54054594039917, 0.9505873322486877, 1.1863160133361816, -0.629498302936554, 2.1058528423309326, 1.393216848373413, 0.19944573938846588, -0.24373407661914825, 0.7386276125907898, 0.525469183921814, 0.20659765601158142, -0.3731047809123993, 0.4885443449020386, -0.5617639422416687, -0.9784547090530396, 0.999408483505249, 0.1917295604944229, -1.4102736711502075, -1.6549224853515625, 0.06587900966405869, 0.7089836597442627, -0.2855285108089447, -2.225774049758911, -0.4486890435218811, -0.4504951238632202, -1.409605622291565, 0.7237160205841064, 1.6385343074798584, 0.3300882577896118, -0.6253290772438049, -1.0998687744140625, -1.035961627960205, 0.0037321753334254026, -0.9701711535453796, 0.9027102589607239, -0.36207321286201477, 0.5315062403678894, 0.8928048610687256, 0.19227208197116852, 1.7859278917312622, 0.6815447211265564, 0.18451166152954102, 1.1000468730926514, -1.0651516914367676, 1.1470458507537842, -0.6428027153015137, 2.4629082679748535, 0.02972641959786415, 2.9250452518463135, 1.725003957748413, 0.07535029947757721], [-0.5184202790260315, 1.920987844467163, -1.0729249715805054, -1.465855598449707, 0.7840716242790222, 0.7951184511184692, 0.5284551978111267, 0.9008349180221558, 0.2654297947883606, 0.7529287338256836, 0.1143670454621315, -0.7144930362701416, -2.5073843002319336, -0.8393400311470032, -1.2564846277236938, 0.024859679862856865, 0.3048613667488098, 0.27071404457092285, 0.32894670963287354, -1.5985251665115356, 0.21610112488269806, -1.0524823665618896, -1.3103737831115723, 0.31037572026252747, -1.4267735481262207, 0.4946342706680298, -0.022881517186760902, -0.10417378693819046, 1.4889172315597534, 2.277595043182373, -0.5827129483222961, -1.7369722127914429, 0.8277069926261902, -1.0077251195907593, 0.4704974293708801, -0.7435917258262634, 0.14339478313922882, 1.4415565729141235, -0.3269256055355072, -0.633690595626831, 0.021049121394753456, 1.0599771738052368, -1.0411427021026611, 0.8745665550231934, 2.4181039333343506, 0.9467174410820007, -0.18149641156196594, -1.386526107788086, 0.729524552822113, -1.0354621410369873, -2.3000168800354004, 0.5121145248413086, 1.5579049587249756, -3.004157543182373, -0.47621437907218933, -1.9434067010879517, -2.367410182952881, -0.3899243474006653, 1.1152526140213013, -0.1600777804851532, 0.509101152420044, -0.8423309326171875, -0.4428739845752716, -0.5170340538024902], [0.49832385778427124, -0.7166000604629517, 0.4462079107761383, 0.6583985090255737, -0.4715837836265564, -1.6075984239578247, 0.8578444123268127, -1.5797721147537231, 1.0357263088226318, -0.8837918043136597, 0.7299797534942627, 1.0931352376937866, -0.6784449219703674, -0.9403095841407776, 0.3871324062347412, 1.1111254692077637, 1.5649186372756958, -1.1687145233154297, 1.368840217590332, 0.6945726275444031, 1.054364562034607, 0.5418366193771362, -0.8728916049003601, -0.5259982943534851, 0.8252617120742798, -0.08178992569446564, 1.3513282537460327, -0.8094411492347717, -0.7709122896194458, 0.343016654253006, -0.3985430598258972, 0.5728946924209595, -1.6852798461914062, 1.1423331499099731, 1.1858470439910889, 1.302100419998169, 1.2962069511413574, -0.504776656627655, 0.30403485894203186, 0.6085569262504578, 1.3305400609970093, 0.8533939123153687, -0.7853487730026245, -0.6948735117912292, 0.9025150537490845, 0.42453476786613464, 0.8284786343574524, 0.16935713589191437, 0.5573803782463074, 0.8592540621757507, 0.013254360295832157, -0.7450209856033325, 0.2836458384990692, -0.011113784275949001, 0.15478834509849548, 0.704262375831604, 1.3470436334609985, 1.1337107419967651, 1.2180867195129395, -1.4311327934265137, 0.05140893906354904, 0.4454288184642792, -0.6223528385162354, -0.3395734131336212], [-1.8061623573303223, 1.231734037399292, 0.49125251173973083, 0.603460431098938, 1.1773289442062378, 1.708582878112793, -0.15700958669185638, -1.3712880611419678, 0.4780292809009552, -0.6956462264060974, 0.7381141185760498, 0.26128828525543213, 0.29951104521751404, -0.12508107721805573, -0.11085162311792374, 2.3090617656707764, 1.1579837799072266, -1.2086793184280396, 0.6895239353179932, -0.6208502054214478, -0.5494652390480042, -0.43182170391082764, -0.6451210379600525, -1.2083362340927124, -0.6282021999359131, -0.2241402268409729, 0.5962024331092834, 0.0660494789481163, 1.4342268705368042, 0.33681273460388184, 0.7230194807052612, -0.580433189868927, -1.1450620889663696, -0.7818160057067871, 0.24767692387104034, 0.3908582627773285, 1.4850404262542725, -0.9944047331809998, -0.7258641123771667, -0.6730329394340515, 1.0094647407531738, -0.2118413895368576, -0.7194314002990723, 0.8149438500404358, 0.1764342039823532, 1.6304540634155273, 0.017115464434027672, -0.44459906220436096, 0.8650187849998474, 0.0399007685482502, -0.25534743070602417, -0.19794408977031708, 0.3030511736869812, 1.0443739891052246, -1.263183355331421, -0.07419724017381668, 0.27236467599868774, -1.8150590658187866, 0.022379010915756226, 1.749029517173767, 2.368350028991699, -0.9648145437240601, 0.8539831042289734, -0.5068193674087524], [-2.427614212036133, -1.3178319931030273, -0.7090600728988647, 0.015345129184424877, -0.6193385124206543, -0.20862869918346405, 0.09984096884727478, 0.8136351704597473, -0.3231752812862396, -0.32996252179145813, 0.01808711141347885, 1.578202247619629, -0.5007285475730896, 1.1771576404571533, 1.0160090923309326, 1.4816704988479614, 1.1653720140457153, -0.3136291801929474, -0.15611660480499268, 0.7106748819351196, 0.4876091182231903, 1.0613311529159546, -0.08217324316501617, -0.6144972443580627, 0.1719944030046463, -0.017400503158569336, -0.4991060197353363, -2.053684949874878, 0.9225065112113953, 0.9048375487327576, -1.1168513298034668, -2.1406171321868896, -0.42282238602638245, 0.47043442726135254, -1.1064422130584717, 0.5589209794998169, 0.9310872554779053, -0.6393248438835144, -0.31608957052230835, -0.5146722793579102, -0.027443718165159225, 0.08306372910737991, -0.3513091504573822, 0.8517105579376221, -0.29047971963882446, -0.05424105376005173, -0.936489999294281, 0.6670318841934204, -0.265826016664505, -1.0995128154754639, 1.4622178077697754, -0.7534403204917908, 1.232882022857666, -2.243220090866089, -0.6694084405899048, 1.3372248411178589, 1.2239081859588623, 0.4453102648258209, -0.14672446250915527, -0.22606879472732544, 0.3903316557407379, -1.8591495752334595, -0.619755208492279, 0.11237446963787079], [-1.1072089672088623, -0.03264695405960083, -0.08602505177259445, 0.16630536317825317, 0.2971111536026001, 1.1486586332321167, -0.7317028045654297, 0.6258195042610168, -1.1435115337371826, -2.0715832710266113, -0.5290334820747375, -0.4033997654914856, -0.19893532991409302, -0.8217931389808655, 0.08859002590179443, 1.3187246322631836, -1.2826982736587524, 0.6849768757820129, 0.5153763294219971, -0.6124741435050964, 0.028845183551311493, -0.6676714420318604, 0.7825971245765686, -0.41475239396095276, -0.3271816372871399, 0.4120100140571594, -0.1018395647406578, -0.7313627004623413, 0.5796000957489014, 0.9675499200820923, 1.0746214389801025, -2.2093029022216797, 1.099169373512268, -0.44200363755226135, 0.4065636694431305, 1.2781184911727905, 1.6570345163345337, -1.388197660446167, -0.7934232354164124, 0.9391418099403381, -0.7070684432983398, -0.7789690494537354, -1.0053634643554688, -0.39728787541389465, 0.45449143648147583, -0.3480100929737091, -0.4250904321670532, 1.2090177536010742, -0.7115143537521362, -0.7009351253509521, -0.2095002979040146, -1.1610041856765747, 0.8275216817855835, -1.7414644956588745, 0.0830250233411789, -0.3455911874771118, 0.32902124524116516, -0.19348961114883423, 1.7830618619918823, -0.43496713042259216, -1.9637120962142944, -0.2245313674211502, -0.9407849311828613, 0.52152419090271], [0.8766346573829651, -0.2639540433883667, -1.8315980434417725, 0.34127193689346313, 0.1938975304365158, -0.709641695022583, 0.4067663252353668, -0.12329287081956863, 0.27575841546058655, -1.153478980064392, -1.2895009517669678, -0.3717041015625, 0.7435611486434937, -1.0904242992401123, -0.32567495107650757, 0.7528202533721924, 0.22798949480056763, -1.7193735837936401, -0.08125864714384079, 1.325832724571228, 1.5692001581192017, 0.3779211640357971, -0.3971253037452698, -2.1039140224456787, 1.62879478931427, -0.5435481071472168, -0.31080445647239685, 1.5985544919967651, 1.1719163656234741, -1.5142617225646973, -2.5322842597961426, 0.7519907355308533, -1.3881967067718506, -0.26543644070625305, 0.2217758744955063, -0.2345975786447525, 0.34866830706596375, 1.0054963827133179, -0.41213831305503845, -1.624579906463623, -2.224689245223999, -0.20246383547782898, 0.010431850329041481, -0.44977810978889465, 0.45262226462364197, -0.4365515112876892, 0.10041123628616333, 0.869342029094696, -0.6938571333885193, -0.1511913239955902, 0.7409141659736633, -1.6746206283569336, 1.941939353942871, 2.0183329582214355, 1.5090500116348267, -0.9570717811584473, 1.3744313716888428, -0.3034474849700928, -0.15535695850849152, 0.32989251613616943, 2.366215944290161, -0.18490095436573029, 0.8825551271438599, -0.8752576112747192], [0.008637605234980583, 0.35479801893234253, -0.8315414786338806, 0.6836827993392944, -0.6450669169425964, -0.9787383079528809, 3.3543436527252197, 0.2798520028591156, 0.4370255768299103, -0.7158092260360718, -0.6104708313941956, 0.9190660715103149, -0.6985162496566772, 0.9868146777153015, 0.9866536259651184, 1.1128787994384766, 0.8983864784240723, -0.564579427242279, -0.8336994051933289, -0.531769871711731, -1.5759170055389404, -1.5333911180496216, 1.0742895603179932, -2.0952882766723633, 0.8222822546958923, -0.807877242565155, -1.7275837659835815, 0.10227242857217789, -0.4591268301010132, -2.2753312587738037, 0.27940237522125244, -1.3576477766036987, 0.7142746448516846, -0.8294523358345032, -0.19820764660835266, -1.4903706312179565, 2.3270037174224854, 0.25414910912513733, -0.48237690329551697, -0.1049400120973587, -0.9764305949211121, 0.7313465476036072, 0.30280864238739014, 1.31890070438385, -0.13092751801013947, -0.23134171962738037, 1.6373766660690308, -0.3773784041404724, 0.609645426273346, 0.9543172717094421, -0.18944352865219116, 0.19407090544700623, -0.17206574976444244, -0.9055128693580627, -0.3373575806617737, -1.6560335159301758, 1.4786382913589478, 1.7771104574203491, -0.5266525149345398, 0.5887625217437744, -0.2293674498796463, 0.5696198344230652, 0.14338968694210052, 1.3441123962402344], [0.8365768790245056, 1.0845685005187988, -0.2723500728607178, 0.2557494342327118, 0.26205793023109436, 0.1272135078907013, -3.6196231842041016, 1.9850648641586304, -0.8536913990974426, -0.2515944242477417, -0.00450197234749794, -0.6018494963645935, 0.29109710454940796, 1.049920678138733, -0.6680680513381958, -0.9395214915275574, 0.9714548587799072, -1.3963227272033691, 0.514385998249054, -0.7830549478530884, 0.32514575123786926, -0.22480465471744537, -0.2808381915092468, 0.1443730741739273, 0.8607740998268127, -0.5947012901306152, -0.022110547870397568, 0.0476604588329792, 1.107814073562622, -0.8633758425712585, -0.5762543678283691, 0.7846428155899048, 0.2978455722332001, -0.5419010519981384, -0.6098073720932007, -0.7412934303283691, -1.2408339977264404, -0.9774571061134338, 1.0983562469482422, -0.15992534160614014, 0.7146462202072144, -1.2995139360427856, 0.07617329061031342, -0.3975333571434021, -2.855328321456909, 1.0686522722244263, 0.5293676257133484, -0.060298483818769455, 0.7774543166160583, 1.474048376083374, 0.33106181025505066, -0.31157204508781433, -0.4209654629230499, -1.0490671396255493, 0.35887616872787476, 0.33720988035202026, 2.523707628250122, 0.8149592876434326, -0.025696642696857452, 0.29636988043785095, -0.8242260217666626, 0.3718291223049164, -0.18665140867233276, -1.2498587369918823], [-1.312118411064148, 1.6826614141464233, -0.5458707213401794, 0.7499073147773743, -1.4670530557632446, -0.44539591670036316, -0.802438497543335, 1.8156096935272217, -0.8344917297363281, 0.618213951587677, 0.9119630455970764, -0.8628684878349304, -0.8735458850860596, 0.39343658089637756, 0.5522047877311707, 0.7114102244377136, -1.537412405014038, 0.11334089934825897, -0.4082093834877014, -1.5049149990081787, 0.627967894077301, 2.7985281944274902, -0.7139710783958435, 0.2644980847835541, 0.4941892921924591, 1.6156141757965088, -1.7518541812896729, 1.9787631034851074, -0.671947181224823, 0.17185552418231964, 0.43560656905174255, 0.09013067185878754, 0.1626739650964737, 1.448679804801941, 1.271195411682129, 2.8598616123199463, 1.6467050313949585, -2.0701725482940674, -0.0021846084855496883, -1.0095303058624268, -0.1424901783466339, -0.40118101239204407, 0.9086810946464539, 0.6722455620765686, 0.44832199811935425, -0.25399452447891235, -0.5997337698936462, 0.1382702738046646, -1.9047073125839233, 0.039069339632987976, 0.34571316838264465, 0.8565669655799866, -0.7112686038017273, 0.1468932032585144, 0.7909371256828308, -1.20225191116333, 1.7184009552001953, -0.3129407465457916, -2.8234803676605225, -0.7900350093841553, -0.8719050884246826, 0.44402363896369934, -0.2307172268629074, 1.4389992952346802], [0.24620667099952698, -0.5663684010505676, 0.5802602767944336, 0.1969451904296875, 0.1301884800195694, -1.477938175201416, 1.1454520225524902, 0.07701878249645233, 0.536224365234375, 2.9262425899505615, -2.0078303813934326, -1.0800621509552002, 1.025588870048523, 1.0860308408737183, 0.7387129068374634, -0.4035627841949463, -0.8365814089775085, -0.14688724279403687, -1.1082011461257935, 0.7951532602310181, 1.290050983428955, -0.5687381625175476, -1.8523029088974, -0.8572818636894226, 0.4854450523853302, 1.8214304447174072, 1.1298637390136719, -2.4912962913513184, 0.2013028860092163, -0.5695155262947083, -0.12041426450014114, 0.7358059287071228, 0.1229630783200264, -0.7907733917236328, 1.645418643951416, -1.5048235654830933, -1.8890587091445923, -2.240821123123169, 0.6344544291496277, -0.32076722383499146, -1.5360596179962158, 0.37902122735977173, -0.08074935525655746, -0.837384819984436, 2.1521356105804443, 0.5353638529777527, 0.8456081748008728, 0.43524807691574097, -0.10259346663951874, -0.12191952019929886, -0.6374907493591309, -2.9607927799224854, 1.5834877490997314, 1.4526445865631104, 0.40357112884521484, 2.2661807537078857, -0.295478880405426, -0.32194289565086365, -2.0752999782562256, -0.8216551542282104, 0.09169209003448486, -0.006022431422024965, -0.26698336005210876, 2.1918447017669678], [-0.39594343304634094, 0.06669554859399796, 2.1618220806121826, -0.5024218559265137, -0.49007147550582886, 1.539409875869751, -0.5624269247055054, 2.1415367126464844, 0.365866482257843, -0.920408308506012, -0.4860193133354187, -0.40783995389938354, 2.0090456008911133, -0.3413875699043274, 0.6606292128562927, -0.7879774570465088, -1.4343253374099731, 0.7468460202217102, -0.7325880527496338, -2.0608139038085938, 1.4719997644424438, -0.670378565788269, 1.6150749921798706, -0.5203002691268921, -0.7678876519203186, 0.2449345886707306, 0.13945287466049194, 1.1841758489608765, 0.6060007810592651, -0.802872896194458, -1.901363492012024, -0.16828811168670654, -0.7818012237548828, -0.4196774959564209, 0.9461663365364075, 1.675583839416504, 0.7685006856918335, 0.7351402640342712, -1.550790548324585, 0.9422605633735657, -0.13687163591384888, 0.00977067369967699, -0.520171582698822, -1.9058961868286133, -0.0223381407558918, -1.4437471628189087, 2.4518024921417236, 0.7001157402992249, 0.9106583595275879, 1.5600718259811401, 2.1831867694854736, -0.6554133892059326, 0.02140483446419239, -0.8151451945304871, 0.4493858218193054, -0.29421424865722656, -0.03549489378929138, -0.4916382133960724, -1.3262395858764648, 0.8075931072235107, -0.8294641971588135, 0.909021258354187, 0.2396114021539688, -0.7835537195205688], [-0.0625484362244606, -0.012999657541513443, -2.2659943103790283, -1.0539337396621704, 0.8528506755828857, 1.394755244255066, -1.4568848609924316, -1.4451724290847778, -0.15322135388851166, 0.1488957405090332, 0.4025675058364868, -0.24575375020503998, 0.38098740577697754, -0.8029560446739197, -0.023610681295394897, 0.6432977318763733, -1.6728363037109375, 0.15142615139484406, -0.23184160888195038, 0.8598962426185608, -0.808997392654419, -0.9635306000709534, 1.3492364883422852, -0.13564342260360718, 2.0205390453338623, -0.6856618523597717, -0.4060136079788208, -0.5669566988945007, 0.4656248092651367, 0.8953217267990112, 0.39493703842163086, 0.9181774854660034, 0.4365142583847046, 1.048188328742981, -0.1349947303533554, -0.9957927465438843, 0.39591923356056213, -0.041430141776800156, -1.2381784915924072, 1.0123300552368164, 1.0974891185760498, -1.1646586656570435, 0.5939961075782776, -0.4450888931751251, -0.8986373543739319, 0.2555437684059143, -0.9150125980377197, 1.007129192352295, 1.692143440246582, 0.1597544550895691, 0.3052133619785309, 0.07894448935985565, -0.2009671926498413, -1.0085318088531494, -0.7762681841850281, -0.9896629452705383, 0.13045234978199005, 1.7121036052703857, 0.41633057594299316, -0.24756842851638794, 1.9076629877090454, 1.3913993835449219, -1.5066804885864258, 1.012184977531433], [0.6212460398674011, -0.8752022981643677, -1.9058915376663208, 0.051781442016363144, 1.5178954601287842, 0.7392595410346985, 0.050826627761125565, -0.5985186100006104, 1.7232173681259155, -0.1501944214105606, 1.2369322776794434, -0.7407951354980469, -0.6784628033638, -0.8640720844268799, 0.5386447906494141, 0.6720779538154602, 0.3062785267829895, -0.00379952066577971, -0.835873544216156, -0.8506150245666504, 0.7421770095825195, -0.5473853945732117, 2.17752742767334, 0.9714159369468689, 2.184384822845459, 1.6192983388900757, 0.25564685463905334, -1.2628555297851562, 0.2349986433982849, -0.5866117477416992, 1.8454992771148682, 0.14426486194133759, -1.2634937763214111, 1.37986421585083, -1.0499975681304932, 1.3905702829360962, -0.4647183120250702, -0.055416714400053024, 0.12279724329710007, 0.8923770189285278, -0.4101773202419281, 0.835063636302948, 0.9853727221488953, -1.1885132789611816, 0.04182664677500725, -1.280039668083191, 0.23136396706104279, 0.5526599884033203, 1.5134438276290894, 0.11094918847084045, -1.513918161392212, 0.06037592515349388, -2.7713682651519775, 0.8426774144172668, 1.6930657625198364, 0.11013009399175644, 0.16950085759162903, -2.034729480743408, -0.6390833854675293, 0.3457300364971161, 0.5013672113418579, 0.3209686577320099, 0.3198935389518738, -0.2250882387161255], [-0.25879615545272827, -0.09552846103906631, 0.9615160822868347, 0.4671988785266876, -0.3135836720466614, -0.4757624566555023, -0.8877483606338501, 2.185960292816162, -1.1141074895858765, 0.581204354763031, 0.2582017779350281, 0.7286883592605591, -0.2724500000476837, -1.0196290016174316, 0.02054809220135212, -0.35589614510536194, -0.6595407724380493, 1.456870198249817, -2.083547353744507, 0.7661616802215576, 2.1629743576049805, -0.8626885414123535, 1.0848445892333984, 0.0648481622338295, 0.8407703638076782, 1.4407023191452026, -0.40250006318092346, -1.9357547760009766, 0.9469944834709167, 1.5785853862762451, -0.27367278933525085, -1.6067842245101929, 0.9535993933677673, -0.7775170207023621, 2.3694310188293457, 1.3881217241287231, 2.4291493892669678, 0.40129101276397705, -0.22500188648700714, 1.0997614860534668, -0.005298217758536339, -1.1906230449676514, 1.8105087280273438, -0.2210044264793396, 2.2550904750823975, 0.05782826244831085, -0.38390493392944336, 0.48085272312164307, 0.47569918632507324, -2.442925453186035, 1.032609462738037, 0.8730564117431641, 0.7932718992233276, 0.3827173411846161, 1.357436180114746, 1.7663978338241577, -1.5379726886749268, 0.9032458066940308, -1.0569108724594116, 0.0933833047747612, -0.7480827569961548, 1.2841269969940186, -0.6055037379264832, -0.1969405859708786], [-0.0019665409345179796, -0.9784687161445618, -1.070285439491272, 2.4505093097686768, 0.009061015211045742, -1.0997371673583984, -0.006304285954684019, 0.7199682593345642, 1.325541615486145, -0.8331382274627686, 0.4454393684864044, 0.4398219585418701, -0.622654914855957, 0.6951857209205627, 0.7093437910079956, -0.5169054865837097, 0.3562416732311249, 0.18342594802379608, -0.8789492845535278, 0.4447386562824249, 1.429539442062378, 0.5901208519935608, -1.5540956258773804, -1.4984270334243774, -1.0625431537628174, 0.4377649128437042, -2.863407850265503, -0.769977867603302, 1.9120521545410156, 0.2793456017971039, 0.21517592668533325, -1.0861670970916748, -2.135920286178589, 0.8515558838844299, 1.1588822603225708, -0.9111661314964294, 1.5859781503677368, -0.2850009500980377, 1.1940163373947144, 1.1492246389389038, 0.03530188649892807, -1.5270248651504517, 1.295590877532959, -1.9163167476654053, -0.7082253098487854, -1.0875675678253174, -1.049835205078125, 1.101433515548706, 1.284875512123108, -0.2907737195491791, -0.30393925309181213, -0.42487233877182007, -0.8087400197982788, 0.6813961267471313, 0.5040563941001892, -1.3805137872695923, -0.8203376531600952, -0.29943326115608215, -2.0037922859191895, 1.1300984621047974, -0.6368275880813599, 0.7170712947845459, 1.1514055728912354, -0.5733556747436523], [0.4897622764110565, -0.2067207545042038, 0.05548557639122009, 0.46732550859451294, -0.35908570885658264, -0.931874692440033, -1.5867393016815186, -0.305342435836792, -1.040644884109497, 0.6438526511192322, -0.645103931427002, -0.48984494805336, 1.6613112688064575, 1.5506938695907593, 1.7268964052200317, 1.8861089944839478, -0.9656556248664856, 1.819117546081543, 0.6595273017883301, -0.06658125668764114, -1.8849198818206787, 0.2709960341453552, -0.4042850434780121, 1.581842064857483, 0.01290438324213028, 0.8453628420829773, -1.0202839374542236, 0.3629075288772583, 0.9942391514778137, -1.058695912361145, -0.44966864585876465, -1.675369381904602, 2.599008321762085, -1.1169683933258057, -0.12393113225698471, -0.31034064292907715, -0.22075818479061127, -0.6583138108253479, 0.27437740564346313, -0.31133103370666504, 0.6847018599510193, 1.1774424314498901, 1.2640049457550049, 1.171445369720459, -1.0592925548553467, 0.7332931756973267, 0.9056021571159363, -1.7731616497039795, 0.7148931622505188, -0.8607653975486755, -0.7738520503044128, 0.9044241309165955, 1.1053425073623657, 0.09180654585361481, -0.4761183261871338, -0.5277530550956726, 1.5737502574920654, 0.7859704494476318, 0.6751777529716492, -0.47097548842430115, 0.784949004650116, -0.5813164114952087, 0.7526044249534607, -0.16684547066688538], [0.06034316495060921, 0.3881226181983948, 0.5725137591362, -0.10543329268693924, 1.4234263896942139, -0.6102874279022217, -0.9146842956542969, 0.8486045002937317, 0.09788133203983307, 0.5947631001472473, 0.3062756657600403, 0.20794129371643066, 0.7448179125785828, -0.6167155504226685, 0.276642769575119, 1.8650072813034058, -2.0227465629577637, -0.5366526246070862, -1.5453848838806152, -1.6840345859527588, -0.5892393589019775, -2.4493050575256348, 0.35385143756866455, 1.0160104036331177, -0.896507978439331, 1.8101246356964111, 1.7725836038589478, 0.7670385241508484, -0.7561127543449402, -0.15140046179294586, 0.6570017337799072, 1.1059048175811768, -1.428383469581604, 0.018435513600707054, 0.7430477738380432, -1.7438639402389526, -1.3745677471160889, -2.379157543182373, -0.25916171073913574, -0.1241421177983284, -1.7922180891036987, 1.8358864784240723, -0.8455607295036316, -0.2542670667171478, -0.7327359914779663, 0.2811299264431, -1.3246132135391235, 0.33871379494667053, -1.3355087041854858, -0.6170101761817932, 0.23771800100803375, 0.4678916037082672, -1.05437171459198, -0.17349617183208466, 0.16848470270633698, -0.11372137814760208, 0.22880463302135468, 1.5267287492752075, 1.311598539352417, 0.38386765122413635, -0.02097579650580883, 0.25873681902885437, -0.30797454714775085, 0.7316352128982544], [-0.8465268611907959, -2.394035816192627, -0.761466920375824, -0.5794336795806885, -0.2298591285943985, 1.5819268226623535, 1.3121939897537231, -1.2975322008132935, 1.1886088848114014, 1.79014253616333, 0.8837277889251709, 0.9362388849258423, 0.19121117889881134, 0.2617394030094147, 0.8624175190925598, -0.3407141864299774, 0.33572810888290405, -0.9740833044052124, 1.0342638492584229, 0.0022118997294455767, -1.699891448020935, 1.1188232898712158, 1.2482173442840576, 1.2983300685882568, -1.437753438949585, 0.2008926272392273, -1.3120214939117432, 0.13680517673492432, -0.6165703535079956, -0.6703336834907532, -0.5197271704673767, -1.0695711374282837, 0.37112510204315186, -0.7196617126464844, 1.1640005111694336, 0.09769821912050247, 0.7426968812942505, -0.5944075584411621, -0.8036415576934814, 0.7384855151176453, -0.7896590232849121, -1.5771313905715942, -1.2509616613388062, 0.11479891091585159, -1.848299264907837, -0.11363670229911804, 1.282759189605713, -1.6766694784164429, 1.275847315788269, -0.16007813811302185, -0.821181058883667, 1.9404135942459106, 0.9673769474029541, 0.8157986402511597, -1.515876293182373, -2.6057605743408203, 1.3935551643371582, 0.04463252052664757, -1.0555864572525024, 0.863743007183075, -1.0685763359069824, -0.69756680727005, -1.4116523265838623, -0.7216078042984009], [-0.49946796894073486, -0.4493439495563507, 1.1901934146881104, 0.08340536057949066, -0.48834937810897827, -0.6341575980186462, -1.3489304780960083, -0.08981525152921677, -1.3705520629882812, 1.0354005098342896, 0.9432746171951294, -1.1515758037567139, -0.9963797330856323, -0.7819734215736389, -1.2340593338012695, -0.029864491894841194, 0.4451540410518646, -0.7850436568260193, -0.41917726397514343, 0.667885959148407, 1.6707556247711182, -0.34956619143486023, 0.21780864894390106, -0.5414561033248901, 1.401221513748169, 0.1921047419309616, 0.11496921628713608, 0.6278228163719177, -0.8515543937683105, 0.3640049993991852, -0.4760752320289612, -0.7607919573783875, -0.4027673304080963, -0.5297476649284363, 1.6174429655075073, 0.42044103145599365, 0.2909085154533386, -0.08310764282941818, 0.6016043424606323, 0.43048539757728577, -0.34325385093688965, 0.21303221583366394, -1.5312204360961914, -0.410748153924942, 1.0911892652511597, -0.22017470002174377, 0.030826430767774582, 2.864779472351074, -0.9063820242881775, 0.3998371660709381, 0.46016329526901245, -0.21734751760959625, -0.0900687426328659, -1.7804806232452393, 0.9432407021522522, 0.2354855239391327, -0.16053292155265808, -0.6450664401054382, 0.6453030109405518, -1.5239994525909424, -0.9511549472808838, 0.8769341707229614, 0.010725627653300762, -0.029799258336424828], [-1.1225770711898804, 0.053428541868925095, -0.5868578553199768, -0.333601713180542, 0.07291046530008316, 0.9611693024635315, -0.8138877749443054, -1.2958266735076904, 0.46726688742637634, -1.8338572978973389, 0.2829899787902832, -1.0718817710876465, 0.5861871838569641, 1.785625696182251, -1.3478410243988037, 0.4700312614440918, -0.025714803487062454, -0.14083455502986908, 0.7304114103317261, 1.8746930360794067, -0.26206308603286743, 1.1356381177902222, -0.08010434359312057, 2.421416997909546, 1.1482269763946533, -1.3298115730285645, 1.4439654350280762, 1.3519413471221924, 0.8800134062767029, -2.0024380683898926, 0.29143574833869934, -1.4443039894104004, 0.3444368541240692, -0.7258986234664917, -0.5455144047737122, -1.2867361307144165, -0.4548051953315735, -0.08522168546915054, -1.1004266738891602, 0.0855269730091095, -0.5838302373886108, 0.14939017593860626, -0.3438419699668884, 0.07292431592941284, 0.4172019958496094, -1.1531085968017578, -1.0215076208114624, 0.20978127419948578, -0.5911047458648682, 1.4970309734344482, -1.3899961709976196, 0.6296494007110596, 1.627236008644104, -1.2562720775604248, -0.6090376377105713, 0.13570822775363922, 0.510606050491333, 0.43807944655418396, 0.04130186140537262, 1.4374865293502808, 0.8278654217720032, -0.6072499752044678, -1.1228281259536743, -0.6303562521934509], [-0.3403899073600769, -1.8297815322875977, -0.5103586912155151, -0.701264500617981, 0.4862532615661621, -0.5602262020111084, -0.7032079696655273, -0.6068568229675293, -0.6900049448013306, 1.0638868808746338, 0.9462177157402039, -0.6134300231933594, -1.8367249965667725, -0.8295514583587646, -0.15662534534931183, 0.2469947636127472, -0.5432732105255127, 1.2211339473724365, 1.177781105041504, -0.006915600039064884, -1.0070393085479736, 0.5719478130340576, -0.037742141634225845, -0.5930001735687256, -0.6625449061393738, -2.5637094974517822, -1.6572695970535278, -0.43443965911865234, -0.05609145388007164, 1.4581457376480103, -0.8194791078567505, -0.28027665615081787, 0.07513834536075592, 0.03821422532200813, 0.8011879324913025, 0.5230374932289124, -0.33632662892341614, 0.047784775495529175, -1.9034346342086792, 0.25613337755203247, 0.3725321292877197, -0.09901632368564606, 0.12875162065029144, -0.8198543787002563, 0.3295649588108063, -0.4264962673187256, 0.5188770890235901, -0.44552621245384216, -1.2077932357788086, 2.004028797149658, 0.25317156314849854, 0.40647807717323303, 0.25618624687194824, 0.609203040599823, 0.4770985245704651, 1.4098658561706543, 0.13646318018436432, 1.386183500289917, -0.026230880990624428, 1.316631555557251, 1.5235278606414795, -0.24654360115528107, -1.0313109159469604, -1.3405307531356812], [-0.004382285289466381, 1.0958434343338013, 1.841639518737793, -0.18184806406497955, -0.24158643186092377, 0.5507464408874512, 1.5952287912368774, 1.1069977283477783, -0.0020954834762960672, -0.5553391575813293, 1.4529732465744019, 0.5177381038665771, 0.5147194266319275, 2.2407267093658447, 0.4988643527030945, 0.35965484380722046, 0.8661909103393555, -1.2014116048812866, 0.7184154391288757, 0.09865860641002655, 0.33490699529647827, -0.7464388012886047, -0.2540217339992523, -0.750112771987915, 0.043878115713596344, 2.4910695552825928, -0.30213889479637146, 1.3798741102218628, 0.8912658095359802, -0.3291454613208771, 0.9529882073402405, 0.8672597408294678, 1.0139906406402588, -0.8398994207382202, -0.085477314889431, -0.4406515657901764, -1.350297212600708, 0.271090567111969, -0.016589585691690445, 1.8658660650253296, -0.7351092100143433, -1.984881043434143, -0.8417940735816956, 0.01273274514824152, -0.47900453209877014, 0.24902725219726562, -1.6473560333251953, 0.61136794090271, -1.71876060962677, -0.32042384147644043, 0.6721623539924622, -1.2320219278335571, -1.2403861284255981, 0.7357423901557922, -0.22278240323066711, -0.2951560914516449, -1.359207034111023, 0.2588801085948944, 0.573972761631012, 0.7159537672996521, 1.0122891664505005, -0.9258174896240234, -0.0247854795306921, -1.0518317222595215], [0.9596716165542603, 0.7456925511360168, -0.2624172568321228, -0.19522520899772644, -0.20026487112045288, -0.6152031421661377, 0.09376756101846695, -0.16872836649417877, 1.0829212665557861, 0.8641896843910217, 0.7918449640274048, 0.13611570000648499, -1.074126124382019, 1.0157324075698853, 2.5016372203826904, -0.9391828179359436, 1.03009033203125, -2.3415348529815674, -0.7973235249519348, 0.462454229593277, 0.6409524083137512, 2.161029577255249, 1.8878700733184814, -1.0111310482025146, 1.5279045104980469, 1.1212043762207031, 0.9333484768867493, -0.2589022219181061, 1.4083595275878906, -0.3327685594558716, -1.263608455657959, 0.4535963535308838, 0.9036681056022644, -1.934969425201416, 1.1278194189071655, -0.32274147868156433, 0.03124140202999115, -1.2518051862716675, -1.156685709953308, 0.2918798625469208, 0.4208620488643646, -0.2602895498275757, 0.9104470610618591, 0.5671576261520386, -1.12861168384552, 1.066500186920166, 0.2642654776573181, 0.4464377760887146, -1.1409144401550293, -1.1695404052734375, 0.8081536293029785, 1.0134345293045044, 1.3596171140670776, -0.7245105504989624, 1.7045705318450928, -0.09941326081752777, 0.37608256936073303, 1.0329259634017944, -0.6354108452796936, 0.5321070551872253, -1.3273112773895264, 1.3870117664337158, -0.4383436143398285, -0.2061455398797989], [1.6055371761322021, -0.8806391358375549, -0.7716262340545654, 1.2112911939620972, 0.995635986328125, -0.6633398532867432, 1.7172199487686157, 0.10339177399873734, -0.08061299473047256, -2.331291437149048, 0.5959380269050598, 2.2576799392700195, -0.008359494619071484, -0.18971794843673706, 2.0641541481018066, 0.5884844064712524, -0.6961302161216736, 0.3985053300857544, 0.730338990688324, 0.43873557448387146, 1.243360996246338, 0.22912606596946716, 0.722446084022522, 0.3838363587856293, -0.5677438378334045, -0.7180084586143494, 0.9004579186439514, -0.7270423769950867, -0.5061075687408447, 0.3650871813297272, -0.8036162257194519, 0.12323939800262451, 0.856349766254425, 1.2176837921142578, -1.591263771057129, 0.4898802936077118, -0.072325199842453, -0.2732158303260803, -0.5472496747970581, -1.100724458694458, -1.0357571840286255, -0.26066046953201294, 1.6824654340744019, 0.445216566324234, 0.279920756816864, -1.6386380195617676, 0.9203630089759827, 0.4033967852592468, 1.5081840753555298, -0.5914860367774963, -2.049527645111084, 0.11919364333152771, 0.1482565551996231, 0.5040450096130371, -0.07696040719747543, -0.581485390663147, 2.136915445327759, 2.030010223388672, 0.6006451845169067, -0.5833570957183838, 0.23127785325050354, 1.0032323598861694, -0.2442760318517685, -1.1851541996002197], [0.2869007885456085, 0.46903103590011597, -0.055611446499824524, 0.4952605962753296, 0.6795550584793091, -0.5821683406829834, 1.656640648841858, -0.03605978563427925, -0.4386465549468994, 0.924731969833374, -0.06522738188505173, -1.2445794343948364, 0.002949231304228306, -0.23717249929904938, 0.0548071414232254, 1.823103904724121, 0.5963039994239807, 2.434434652328491, 0.12282464653253555, 0.664891242980957, 2.3397865295410156, -2.234851121902466, 0.8577094674110413, 1.1230181455612183, -0.5189197659492493, 0.28843483328819275, 0.7032977938652039, 0.21838313341140747, 1.3638499975204468, -0.06654098629951477, 1.4442440271377563, -0.9099597930908203, 0.563244640827179, -0.5713322162628174, 0.28347882628440857, 0.5622408390045166, -0.5630451440811157, 0.1983165740966797, -1.6655572652816772, 0.5860868692398071, -1.5464321374893188, -0.1931004375219345, 0.5819276571273804, -0.09781146794557571, -0.15340159833431244, -0.11981355398893356, 0.05262446030974388, -0.8629523515701294, -0.07217264175415039, -1.0570075511932373, -0.6694806814193726, -1.8330492973327637, -1.2324857711791992, 0.15758894383907318, -0.2649862766265869, 0.39980602264404297, 0.3404289484024048, -0.5801947712898254, 0.5720647573471069, 0.04975772649049759, 1.212295413017273, 1.1372883319854736, -0.9318459630012512, -0.04428994283080101], [0.005488080903887749, -0.20462706685066223, -1.29257071018219, 1.564001202583313, -0.20898188650608063, 1.264133334159851, -0.5209860801696777, -0.8172829747200012, -0.3969455063343048, 0.22088488936424255, -0.7903828620910645, 1.454390525817871, 2.6841187477111816, -0.8808274865150452, 0.5587365627288818, 0.680498480796814, 1.1080565452575684, -0.9977162480354309, -0.8654423952102661, -1.703326940536499, -0.24441371858119965, 0.39878150820732117, -0.07767152041196823, 0.6172457933425903, 0.880667507648468, 1.774471402168274, 2.334033966064453, -0.011144530028104782, 0.252788245677948, 0.44707605242729187, -1.1454989910125732, 0.7063561081886292, -0.9801476001739502, -1.2386856079101562, 0.9712230563163757, -1.7283474206924438, -0.12288282811641693, -0.29544106125831604, 0.31822168827056885, -0.6901208758354187, -0.3169315755367279, -0.41808170080184937, -0.5708471536636353, 1.4022953510284424, 0.31753626465797424, -0.7994635701179504, -0.31791144609451294, 1.2465342283248901, 1.0166031122207642, 0.5967865586280823, -1.5561320781707764, -0.03271764889359474, -0.963415801525116, -1.518808126449585, 0.09548753499984741, 0.10968652367591858, -0.18133224546909332, -0.11192887276411057, 1.0431404113769531, 2.199462890625, 0.4469441771507263, 1.3286359310150146, -0.22595256567001343, -1.2690333127975464], [-0.08690021187067032, -1.2262418270111084, 1.211051344871521, -0.5694611072540283, -0.10799451172351837, -0.1545913815498352, -1.4677928686141968, -0.36109331250190735, -0.5366037487983704, -0.05118994787335396, 1.1313520669937134, -0.4083104729652405, 0.4028667211532593, -1.9868431091308594, 0.08174320310354233, 0.13070499897003174, 1.2409342527389526, 0.7057419419288635, -0.08671104162931442, -0.8964844346046448, -1.5448318719863892, 1.683283805847168, 0.17195452749729156, -0.9497539401054382, -0.034398555755615234, 0.08974872529506683, -0.3385384976863861, 0.46887022256851196, 1.515061616897583, -0.9251443147659302, -1.5706570148468018, -1.3064470291137695, 0.6901385188102722, 0.7384805679321289, 0.28027427196502686, 1.1278011798858643, 1.0427664518356323, 0.5308243036270142, -0.27805382013320923, -1.0586612224578857, -0.2269459068775177, 0.2750476896762848, -0.1952153891324997, -0.5510216355323792, 0.04921398684382439, 2.3277335166931152, -0.6051152944564819, -0.576968252658844, 0.6093202829360962, 1.748060703277588, -1.3436647653579712, -1.8115341663360596, -0.6811707019805908, 1.0890649557113647, 1.3842096328735352, -0.5683032870292664, 0.43812844157218933, 0.6099885702133179, -0.43706128001213074, -0.28030702471733093, -0.1921466588973999, 1.086051106452942, -1.0919979810714722, -2.0214664936065674], [0.9882599115371704, 0.10544675588607788, -0.43627190589904785, 0.36889025568962097, 0.4259253144264221, 0.12000659108161926, -0.6718994975090027, 1.1602566242218018, -0.491832435131073, 0.956323504447937, -0.5634938478469849, 0.9120395183563232, -0.9215449690818787, 1.1703013181686401, -0.257548987865448, 0.27349671721458435, 0.4511101543903351, 0.2564687132835388, 0.2255111187696457, 0.13230101764202118, -0.9892821311950684, 0.3647668957710266, -0.15910567343235016, -0.028004927560687065, -1.0607244968414307, -0.6547459959983826, -0.23465660214424133, -1.265318751335144, -0.36203262209892273, -0.9444841146469116, -1.0301132202148438, 0.19886325299739838, 0.8536033034324646, 0.07663044333457947, 1.969353437423706, 0.21502947807312012, -0.5283802151679993, -0.618270754814148, 0.2502840757369995, -0.13501977920532227, -0.263461709022522, -0.7580548524856567, -1.5362995862960815, -1.0787628889083862, 1.168724775314331, -0.9946966171264648, 1.8091092109680176, 0.5785566568374634, -1.6825214624404907, -0.335323303937912, 0.8242163062095642, -1.4755761623382568, -1.1740425825119019, 0.3774034380912781, 0.8882761001586914, -1.6267614364624023, 0.4491449296474457, -0.10352889448404312, 1.280405044555664, -0.5211517810821533, 0.9080514907836914, 1.7117364406585693, 0.1297670155763626, -1.406212568283081], [0.8132300972938538, 0.9487829804420471, 0.4053211510181427, 0.6274535059928894, -1.0773932933807373, -0.25674694776535034, 0.554924726486206, -0.9468638300895691, -0.18467676639556885, 1.7113533020019531, 0.3490715026855469, 0.1213102638721466, -0.5635495185852051, -2.0515213012695312, -0.20155929028987885, -0.1366739720106125, -0.2997271716594696, -0.12266229838132858, -1.310892939567566, 1.487881064414978, -0.35584741830825806, -0.2727554738521576, -0.3440224528312683, 0.6235123872756958, 0.09408272057771683, 0.5241201519966125, -0.38122156262397766, -1.7421247959136963, -0.2833038568496704, -1.1533743143081665, 0.9107477068901062, -0.7017074227333069, 0.579403817653656, -0.10099002718925476, -0.5241286754608154, 0.11700082570314407, 0.9145407676696777, 1.9249789714813232, 0.32791587710380554, -0.014175718650221825, 0.5432048439979553, -0.5735583901405334, 0.5911912322044373, 0.15704691410064697, -0.4259549379348755, -0.17906901240348816, -0.015278373844921589, -0.876961350440979, -0.6050955653190613, 1.6343649625778198, -0.12538230419158936, 0.7101086974143982, 1.3991479873657227, -0.0935928001999855, -1.294419765472412, 0.33464136719703674, -1.896472454071045, 1.2718945741653442, -1.4063668251037598, 0.26246365904808044, 0.3187439739704132, -1.3285962343215942, 0.29006054997444153, 0.5852748155593872], [-0.7503674626350403, 0.6864720582962036, 2.202732801437378, 0.4181964695453644, -0.30425241589546204, 0.18368016183376312, 0.12253345549106598, 0.12463436275720596, 1.261508584022522, 0.42273733019828796, -0.19932736456394196, -1.0098540782928467, 1.3357683420181274, 0.48977088928222656, 0.5048698782920837, 0.19523276388645172, 0.22907637059688568, -1.2625033855438232, -1.359488606452942, -0.4839014410972595, -0.27641040086746216, -0.5907296538352966, -0.25992050766944885, 0.3617636561393738, 0.193257138133049, 0.15048664808273315, -1.2092885971069336, 0.6079906225204468, 0.5847784280776978, -0.3325369954109192, 0.05195139721035957, -0.3368866443634033, -1.0246464014053345, -0.0759289339184761, 0.22737495601177216, -0.14576931297779083, 1.7092951536178589, 0.0629565417766571, -0.38833069801330566, -0.9431291222572327, 0.6667909622192383, 0.11172759532928467, 0.7808660864830017, -1.0460052490234375, 0.34878984093666077, 0.4813832938671112, 0.28573140501976013, 0.017039617523550987, 1.0684398412704468, -1.022933006286621, 1.9257301092147827, -0.285706102848053, 0.8869782090187073, -0.5532870888710022, -0.9760870933532715, -0.20585019886493683, -0.04987851902842522, 0.34724411368370056, 2.1781508922576904, 0.5898051857948303, 0.2922994792461395, 1.400472640991211, 0.735332727432251, 0.4101777672767639], [-0.11849802732467651, 0.2768102288246155, 0.5897451043128967, 0.5609803795814514, 1.4125642776489258, 2.1311044692993164, 0.4866437315940857, 0.0895543247461319, -2.040012836456299, -0.4201396405696869, -0.04683326929807663, -1.4790910482406616, 1.246774435043335, -0.0774867832660675, -0.4735576808452606, -0.7789645791053772, -2.1275360584259033, 1.0412847995758057, 1.4152570962905884, -1.292152762413025, 0.1757502406835556, -0.008699048310518265, -0.6861392855644226, 0.3840232789516449, -0.9938468933105469, 1.4823881387710571, 0.9360936284065247, -0.6828793287277222, -1.012995958328247, 0.9889066219329834, 0.21678315103054047, -0.9822291731834412, 1.1171398162841797, 1.451397180557251, -0.9064986109733582, -0.08701921999454498, 0.42196595668792725, -1.4503611326217651, 0.5322604179382324, -0.09263557940721512, 0.09154145419597626, 0.2721015512943268, -0.3483220338821411, 0.49915724992752075, 0.7750251889228821, 0.5034170746803284, -1.675182819366455, -0.8218420743942261, -1.4506585597991943, 0.834682047367096, 0.8100850582122803, -1.6452335119247437, -0.31603312492370605, -0.5968435406684875, -0.41143855452537537, -1.8689872026443481, -0.6838947534561157, 1.180052638053894, -1.501114010810852, 1.4942584037780762, 0.9309446811676025, 0.15579895675182343, -0.06212785467505455, 0.1314765214920044], [0.6089162230491638, -2.461852788925171, 2.013538122177124, 0.17281094193458557, 0.5330832600593567, 0.16776995360851288, -0.09802784025669098, 0.1458524912595749, 0.38249480724334717, 1.3113213777542114, 1.8522415161132812, 0.26615190505981445, -0.037750229239463806, 1.6408181190490723, -1.272410273551941, -0.8563171029090881, 0.21445544064044952, 0.797491729259491, 0.32063189148902893, -1.1678917407989502, 0.6605427861213684, 0.2729036509990692, 0.6161201596260071, 1.6663498878479004, -0.29368725419044495, 0.4468889832496643, 1.4231886863708496, 1.7987233400344849, 1.1989805698394775, -0.6651548743247986, 0.6598376035690308, 0.27748119831085205, 0.03645114600658417, -1.444589376449585, 1.2633602619171143, -1.0664423704147339, -0.9575674533843994, -0.13227276504039764, 0.16358168423175812, -0.8747246861457825, -1.0317516326904297, 0.3961034417152405, 1.2342352867126465, 1.7692105770111084, 2.0349955558776855, -0.28363415598869324, -2.0257651805877686, 0.3319839835166931, -1.2371742725372314, 1.2892557382583618, 0.14020366966724396, 0.557469367980957, -0.33084186911582947, -0.11636759340763092, -0.15585722029209137, 2.7674195766448975, 1.582698106765747, 0.440049946308136, 0.33983203768730164, -1.1854429244995117, 1.2621468305587769, -0.7564300894737244, 0.782121479511261, 0.8115332126617432], [-0.21152658760547638, 0.3297967314720154, -0.525461733341217, -1.727561593055725, -0.9073801636695862, -0.44840556383132935, 1.0285828113555908, 0.9792051315307617, -0.10016482323408127, -0.7402690649032593, -2.4725558757781982, -1.2444239854812622, -0.9228845834732056, -0.8145239949226379, 0.4950964152812958, 0.11312204599380493, 0.9864782094955444, -0.3384809195995331, 0.21474500000476837, -0.1420474797487259, -0.2790306508541107, 1.0040678977966309, 0.7994933128356934, 0.9395318627357483, -0.36714547872543335, 1.9434027671813965, 0.46082383394241333, -0.7112146019935608, 0.3609775900840759, 0.37595996260643005, -1.15931236743927, -0.7788689136505127, 0.235487163066864, -0.4361094832420349, -0.44403111934661865, 0.8447333574295044, -0.7959197759628296, 0.021859314292669296, 1.3143736124038696, 0.2554492950439453, -0.18071193993091583, 1.246761679649353, 1.3346045017242432, 1.8101553916931152, -0.5049076676368713, -0.1418638974428177, 0.10605479031801224, 0.2875349521636963, -1.4099905490875244, 1.8832910060882568, -0.5028142333030701, -0.0302176084369421, -0.6258669495582581, 0.08334513753652573, -0.7868343591690063, -1.0145628452301025, -1.1952000856399536, -1.0586152076721191, -1.0093772411346436, 0.39555031061172485, -0.6914772987365723, 0.30967292189598083, 0.4201250970363617, 0.7961379885673523], [0.6994916796684265, 0.9702456593513489, -0.24979360401630402, -0.4963112473487854, 0.9460223913192749, 0.7683855891227722, -0.7024635672569275, 0.44856905937194824, 1.4230536222457886, 1.105624794960022, 1.5043753385543823, -0.6817977428436279, -0.9955641627311707, 0.5263530015945435, 0.029421018436551094, -0.18617048859596252, -0.1823287308216095, -0.5777968764305115, -0.15450334548950195, -1.592301368713379, -1.2411457300186157, 0.14941442012786865, -1.0396442413330078, 0.12638261914253235, -0.25303301215171814, -0.2143699675798416, 1.851507544517517, -0.021245013922452927, 0.034134164452552795, 0.6240199208259583, 2.133821487426758, -0.3074929118156433, 0.6732383370399475, -1.7189174890518188, -1.3462220430374146, -1.0163449048995972, 1.3472319841384888, 0.2378203570842743, -0.08575081825256348, -0.4272104501724243, -1.2540754079818726, -0.1361648142337799, -0.017358126118779182, -2.0378880500793457, 1.4199234247207642, 0.39291298389434814, 1.215519666671753, -0.09956616163253784, -1.3062517642974854, 1.5060592889785767, 2.7534384727478027, -1.8933273553848267, 0.8648945093154907, 0.9943579435348511, 0.5789355039596558, -0.03193558380007744, -0.25908470153808594, 1.4821839332580566, -0.12611572444438934, 0.30021631717681885, 0.24151569604873657, 0.7556941509246826, 1.5368444919586182, -2.3452372550964355], [0.4478286802768707, -0.0042678858153522015, 1.3817954063415527, 0.10309648513793945, 1.6633894443511963, 0.7117651700973511, -0.4229053556919098, -0.10952632874250412, -0.3343740701675415, 0.7674635648727417, -0.11177311092615128, 0.7621040940284729, -0.2803844213485718, 0.2715328335762024, 0.12377233058214188, 1.4803615808486938, -0.2964710295200348, 1.2159662246704102, -0.36478862166404724, -0.47012439370155334, -1.12006413936615, 0.9794635772705078, -0.327690452337265, 0.7950421571731567, -1.8723232746124268, -0.7363636493682861, -0.7019987106323242, -0.007601600140333176, -0.3869533836841583, 0.55782151222229, 0.0404575951397419, -1.1458123922348022, 0.15577180683612823, 1.5033669471740723, -0.5688976049423218, 0.10428232699632645, 0.6843939423561096, 0.11461582034826279, -0.21872904896736145, 0.392890065908432, -0.0853840634226799, 0.46064049005508423, 1.8519595861434937, -0.08183932304382324, -0.3256898820400238, 0.09095147252082825, -2.077030897140503, 0.6200793981552124, 1.302173376083374, -0.5186624526977539, -1.1982442140579224, 1.1773771047592163, -1.4781126976013184, 0.10045934468507767, 0.10218465328216553, 0.9285320043563843, 0.667691707611084, 1.7781919240951538, -0.26141276955604553, 0.1334368735551834, -1.4762412309646606, -0.6361374258995056, 0.8411184549331665, -0.8557792901992798], [0.758493959903717, 0.8049957752227783, -1.0088374614715576, 2.769192934036255, 0.04331548139452934, 0.4632948338985443, -0.8285060524940491, 0.904893159866333, 0.7940905094146729, -0.16383229196071625, 1.142815351486206, 0.5812646746635437, 0.11987325549125671, -0.40573230385780334, 0.652619481086731, -0.7508118748664856, 1.38583242893219, 0.2054458111524582, -0.9892781972885132, 0.5495171546936035, 0.8830710649490356, 0.12753428518772125, 0.5839109420776367, -0.28586021065711975, -0.2184687852859497, 1.2424242496490479, 1.9295939207077026, -1.6382646560668945, 0.1739395260810852, -0.7556330561637878, -1.5561823844909668, -0.05537048354744911, -0.3906362056732178, 0.4158180058002472, -0.2779785096645355, -0.9816251397132874, 0.21636289358139038, 0.5965325832366943, -1.0038992166519165, 0.238547220826149, 0.4491361379623413, -1.8372727632522583, -0.5720112919807434, 0.6103429794311523, -0.8223633766174316, -0.42278537154197693, 0.43297892808914185, -0.3704179525375366, -0.7769666910171509, -0.082007497549057, -1.576561689376831, 0.005389688070863485, 2.6270229816436768, -0.9543880224227905, -0.8169654607772827, -0.48041343688964844, -0.06102338433265686, -1.8146493434906006, 0.7213018536567688, -0.19685697555541992, -0.6349883675575256, -1.42343270778656, 0.20777633786201477, -0.19911804795265198], [-0.8047383427619934, 1.058485984802246, -0.9117433428764343, 0.4228213429450989, -0.6839606165885925, 0.05841903015971184, 2.833502769470215, 0.12667898833751678, 0.6627740859985352, -0.13760720193386078, -0.24349366128444672, -0.46971866488456726, 0.39767351746559143, 1.1209124326705933, 0.5841531157493591, 0.26186519861221313, 1.4977060556411743, -1.3040876388549805, 0.862462043762207, -0.04509785771369934, 1.2099748849868774, -0.6153689026832581, 0.042413920164108276, -0.5354891419410706, 0.9083573222160339, 1.2108545303344727, 0.49471479654312134, -0.3248080611228943, -1.0691964626312256, -0.9450125098228455, 0.5452191233634949, -0.21120958030223846, -1.511783480644226, 0.9379264116287231, 0.08074021339416504, 0.15229959785938263, -0.7193419933319092, 0.28033196926116943, -0.901418924331665, -1.1178280115127563, -1.7241907119750977, 0.9375256896018982, -1.0409742593765259, -0.34299513697624207, 0.17995205521583557, 0.44380855560302734, 0.30947762727737427, 0.26400089263916016, -0.9301425218582153, 0.6342229247093201, 1.8006144762039185, 0.958932638168335, -1.7140101194381714, -0.9308484792709351, 0.2609460651874542, 0.3339238464832306, 1.0839943885803223, -0.5511782169342041, 0.5072086453437805, -1.414069414138794, 0.17765049636363983, -0.40229469537734985, 0.22199110686779022, 1.234821081161499], [0.08474740386009216, -0.6480118632316589, 0.6469545960426331, -0.08618563413619995, -0.7784078121185303, 0.07582754641771317, 0.03603212535381317, -1.941564917564392, 0.053516268730163574, 0.4816591441631317, 1.3802720308303833, -0.2613930404186249, -0.9330250024795532, -0.41334620118141174, 0.5445202589035034, 0.2845718562602997, -1.016982913017273, 1.767320156097412, -1.2721577882766724, -0.7634070515632629, 1.3542957305908203, 0.7436807155609131, -1.8529728651046753, -0.4443177282810211, -2.232544422149658, -0.5750833749771118, 0.3378181755542755, 0.04227062314748764, 0.11727763712406158, -0.9121450781822205, 0.35829323530197144, -0.17078429460525513, -0.20768612623214722, 1.5694012641906738, -0.969871997833252, -0.07784418761730194, -2.154275417327881, 0.5146703720092773, 0.9085695743560791, -0.9283798336982727, -0.9210101962089539, -1.021957516670227, -1.1962029933929443, 0.4861892759799957, 0.6684327125549316, 0.9257978200912476, -0.6350078582763672, -1.3983805179595947, -0.06022457405924797, -2.246701955795288, -0.29312723875045776, 1.908629298210144, 2.0398964881896973, 1.2652968168258667, 0.7593405842781067, -0.22532707452774048, 0.1532512903213501, -0.5058542490005493, -1.213397741317749, 1.1294312477111816, -1.7544678449630737, 1.4326783418655396, 0.8565066456794739, 1.8691315650939941], [-0.20032134652137756, -0.007592878304421902, 1.460289716720581, -0.6762301325798035, 1.6454821825027466, -0.06253021955490112, -0.6968530416488647, 0.15532463788986206, -1.6065407991409302, -0.8921393156051636, -0.0032749834936112165, -1.835682988166809, -1.2338967323303223, 1.0685302019119263, -0.41470202803611755, 0.011314869858324528, -0.0519874207675457, 0.7424303889274597, -0.5156480669975281, 0.4128238260746002, -1.4903322458267212, 1.4840822219848633, -0.2465198040008545, -0.44547557830810547, -1.2383407354354858, 0.7039937973022461, -1.1957703828811646, 0.8139883875846863, -0.8549228310585022, 0.07579421997070312, -0.43054744601249695, -1.7924487590789795, 1.173695683479309, -0.1360611766576767, -1.9318114519119263, -1.1311936378479004, 0.7821021676063538, 0.6044481992721558, 1.9409911632537842, -0.14426761865615845, 1.8838343620300293, 0.513293981552124, 0.9747015237808228, -0.3472883105278015, 1.1894068717956543, -0.33736133575439453, 0.9349358677864075, -1.022511601448059, 0.5035302639007568, -0.0008918268140405416, 0.5675597786903381, 0.9973604679107666, -0.4792908728122711, 0.7288494110107422, 0.1310081034898758, -1.5021933317184448, -0.5301010012626648, -0.6657028794288635, -0.5673041939735413, -0.46349868178367615, -0.21067634224891663, 1.6361955404281616, 0.7555095553398132, -0.5207034349441528], [-0.5739823579788208, -0.5275408625602722, 0.8283061385154724, -0.6492608785629272, -0.543667733669281, -0.39537569880485535, 1.2467021942138672, 0.5146543979644775, 1.2711533308029175, -0.8328032493591309, 0.7948964238166809, 0.453148752450943, -0.3511366546154022, -0.5890495777130127, 0.05912762135267258, 1.2615492343902588, -0.7880482077598572, -0.8170978426933289, 0.7188851237297058, 0.14604716002941132, 0.5579861402511597, -0.049138713628053665, 0.27243560552597046, -1.5962435007095337, -0.11239422857761383, -0.3549966514110565, 0.6070990562438965, 0.12843607366085052, 0.7967673540115356, 0.5554351806640625, 0.1394268423318863, 0.2000320553779602, -1.3031946420669556, -0.7792887091636658, -1.8993207216262817, 1.045964241027832, -2.2927002906799316, 0.24613462388515472, -0.48246145248413086, 0.7691165804862976, 0.31060853600502014, 1.6393392086029053, 1.4753527641296387, -0.05143357813358307, 0.011099784635007381, -0.3159576952457428, 0.9661126136779785, -0.9542990326881409, 1.067646861076355, -0.2065618932247162, -0.5202022194862366, 1.2454982995986938, -0.95828777551651, 2.2816414833068848, -0.13925030827522278, -0.46171316504478455, -1.945953130722046, -1.7038555145263672, -0.4861239790916443, -0.05768151581287384, -2.0149056911468506, -1.057056188583374, 0.9570499062538147, -1.08991539478302], [-0.6732078194618225, 0.41738730669021606, -0.8776694536209106, 2.6164510250091553, 0.8363919258117676, 0.13426929712295532, -1.7116495370864868, -0.3451295495033264, -0.7970772981643677, 0.44312769174575806, -0.5946826934814453, -0.254536896944046, -0.5971482396125793, -2.295095443725586, -0.34958311915397644, -0.6735725998878479, -0.0699736699461937, 0.06224915385246277, -0.8890355825424194, -0.5259082317352295, 0.19593070447444916, 0.548330545425415, 0.9690600037574768, -0.7812192440032959, 0.7628934979438782, 0.575366199016571, 0.6180338263511658, -0.1794745922088623, -1.4658807516098022, 0.44112643599510193, -0.5240417122840881, -0.5478639602661133, 0.6763243079185486, 0.9273837804794312, -0.23438625037670135, -1.088248372077942, -1.252782940864563, 0.545444130897522, 1.3703510761260986, -0.16944371163845062, 0.49842140078544617, -0.8195029497146606, 1.0363194942474365, 0.4928340017795563, 1.7962533235549927, -0.9707819223403931, -1.738848328590393, 0.08697764575481415, -0.9070027470588684, 0.15254119038581848, 0.16294360160827637, -1.226313829421997, -0.1815425306558609, 0.2012973129749298, 0.8964939117431641, -2.3578786849975586, -0.6285586357116699, -0.6522374749183655, 2.4950432777404785, 0.5181496739387512, 0.049221254885196686, -0.10984835028648376, 0.9187464118003845, -1.1523652076721191], [-1.7767959833145142, 0.01891832798719406, 0.5865931510925293, -0.34515830874443054, 1.8031946420669556, -0.6576172709465027, 1.931684136390686, 0.10055215656757355, -0.16452984511852264, 1.144219160079956, -0.5364293456077576, -1.0160108804702759, -1.2896922826766968, 1.7353284358978271, -0.3977980613708496, 0.03784371167421341, 0.963489294052124, 0.7378312349319458, 0.5089971423149109, -1.8857479095458984, -0.7276996374130249, 0.8168725967407227, -0.5183032155036926, -0.9425963759422302, 0.06435553729534149, 0.18861642479896545, -0.07324612885713577, -1.1802257299423218, -2.3763480186462402, -0.40589025616645813, 0.1018187403678894, 1.2309634685516357, -0.5697096586227417, -0.017050735652446747, 1.1996325254440308, 0.04963064566254616, 0.5775882005691528, 0.6516122817993164, -0.4933747351169586, -2.180105447769165, 0.12667019665241241, 1.0331231355667114, 0.10167286545038223, -0.7123910188674927, 0.2945731580257416, -0.134803906083107, 1.7007381916046143, -1.1079576015472412, -1.1049120426177979, 0.5797057747840881, -0.5874487161636353, 1.4806638956069946, 0.8794893622398376, -0.36053624749183655, -0.2552310824394226, 1.3915823698043823, -0.23935148119926453, -1.955263376235962, 1.146440863609314, -0.7463716864585876, 0.5304073095321655, -0.05192891135811806, -0.656910240650177, 0.9654644131660461], [-0.9918707013130188, 1.0223252773284912, 1.095047950744629, -0.13998737931251526, -0.15820817649364471, 1.8869104385375977, 0.15866947174072266, -0.20209430158138275, -0.9717894792556763, 0.8489732146263123, -0.5691273212432861, 1.061352252960205, 0.7976765632629395, 0.1107931137084961, 0.5192500352859497, -0.22590264678001404, -0.7429386377334595, 0.19019655883312225, 0.49177423119544983, 1.4679467678070068, -1.2326518297195435, -0.058622296899557114, -0.6226820945739746, -1.0623236894607544, -1.0692274570465088, -0.1934405416250229, 0.6973842978477478, -0.010379276238381863, -0.5864549279212952, 0.37543079257011414, -0.874518096446991, -0.722011923789978, -0.06644687801599503, -0.13824550807476044, 0.6061531901359558, -0.7196629047393799, 1.7850219011306763, -1.1232885122299194, 1.9314543008804321, -0.12679290771484375, 0.9574965238571167, -1.4500296115875244, 1.8228371143341064, 1.5349262952804565, 1.8105987310409546, 0.8005343079566956, -0.3033551275730133, 0.5485048890113831, -0.675280749797821, -0.41533905267715454, -1.468765139579773, -0.5154306888580322, -0.6569089889526367, -0.4561532139778137, 1.3175138235092163, -0.11194591224193573, -1.2955759763717651, -0.15651902556419373, -0.23679719865322113, -2.2533011436462402, 1.209620714187622, -0.23586320877075195, -0.13618284463882446, 0.33137258887290955], [-1.1200288534164429, 0.740354597568512, 2.6966516971588135, -0.030409619212150574, 2.530261516571045, 0.9021815061569214, -0.4397314190864563, 0.2056608945131302, -0.39825350046157837, -0.01290613692253828, 0.32622474431991577, -0.9018673300743103, 0.16161610186100006, -0.5751014351844788, 0.10741938650608063, -0.6090455651283264, 2.3061981201171875, -1.281814694404602, -0.5909237861633301, 1.1238220930099487, -0.4131563901901245, 0.8698064088821411, -1.324662208557129, 0.8003790974617004, -0.9884707927703857, 2.1816256046295166, 2.1277358531951904, -1.2710331678390503, 1.059045672416687, -1.19805908203125, -0.43453049659729004, 0.016754062846302986, -0.573443591594696, 0.33686038851737976, -2.1889851093292236, -0.23349006474018097, 0.369297057390213, -0.0978187844157219, -0.11078125238418579, 1.1855748891830444, -1.3097212314605713, -0.12466254830360413, 1.2622414827346802, 0.3971151113510132, 0.4058518707752228, 1.579193115234375, -0.852996289730072, -0.5669392347335815, -0.17863492667675018, 1.1397144794464111, -1.327868938446045, -1.3331471681594849, -1.119279146194458, 0.667493999004364, -1.539772868156433, -0.09194279462099075, 1.3984647989273071, -1.881571650505066, 0.6808372735977173, 0.09389343857765198, 0.6391811966896057, -0.6765631437301636, 0.2301560640335083, 0.23017938435077667], [0.29495754837989807, -0.8149417042732239, -0.5012632608413696, -0.6378375887870789, -0.18506933748722076, -0.9516139626502991, -1.8077386617660522, 0.9238006472587585, 0.45776137709617615, -0.726051390171051, -0.7041312456130981, 1.9769599437713623, 0.37733912467956543, 1.7354494333267212, -1.149677038192749, -1.7131977081298828, 0.03270339220762253, 1.0557773113250732, 0.31570908427238464, -0.38707834482192993, 0.6018363833427429, -1.108282446861267, 0.31724926829338074, -1.8031271696090698, -1.2346147298812866, -0.1281852424144745, -1.2245336771011353, -0.0750223845243454, -0.07559806108474731, -0.9148051142692566, -0.0028461224865168333, -0.44644486904144287, -0.051823556423187256, 0.19248011708259583, -1.2116129398345947, -0.5991045236587524, -0.2827504277229309, -0.907402515411377, -1.5776442289352417, -0.33500227332115173, 0.38088902831077576, 0.3677065670490265, 1.5516328811645508, 0.8140838742256165, 1.0952128171920776, -0.6449223160743713, 0.2595885396003723, -1.0246376991271973, 0.5954951047897339, 1.080237627029419, -1.5412075519561768, 0.651961088180542, -0.3953748941421509, 1.172373652458191, 0.8880814909934998, 0.7938054800033569, 1.4977201223373413, -1.3402553796768188, -1.8954966068267822, 2.032362937927246, -0.9138695001602173, -0.02217641845345497, 0.10181108862161636, -0.3442073166370392], [-1.0836730003356934, 0.9572159647941589, -0.410017728805542, -0.37192976474761963, -0.7065661549568176, 0.03777139633893967, 0.27917638421058655, 0.23085246980190277, -1.5128750801086426, -1.209333896636963, -2.891935110092163, 1.8631318807601929, 0.3195585310459137, -2.0220301151275635, -0.03176750987768173, 0.2567391097545624, 0.6852036714553833, -0.39241209626197815, -1.3501509428024292, -1.7709952592849731, 0.5524391531944275, 0.3427579998970032, 1.5611836910247803, -1.2825614213943481, 0.12135544419288635, -1.7519959211349487, 1.1817659139633179, 1.4605742692947388, -1.6686811447143555, 2.1042983531951904, -1.3488284349441528, -0.9654327034950256, -0.06983254104852676, 1.269394040107727, 0.09709412604570389, 0.17622481286525726, -2.083446741104126, 0.12235864251852036, -0.743096113204956, -0.5408982634544373, 0.5137905478477478, 0.14304804801940918, 2.6268107891082764, -0.0421915203332901, -1.0224733352661133, 0.2226148098707199, 1.676324725151062, 0.5504092574119568, 1.5519883632659912, -1.2378475666046143, 0.4836949110031128, -0.13075999915599823, -0.04021083936095238, 0.2293289601802826, -1.4978188276290894, 0.8487765192985535, -0.9087111353874207, 0.19783709943294525, 0.7345818281173706, -2.1150243282318115, -1.0319240093231201, -0.1761566698551178, 0.6862099170684814, 0.27951276302337646], [0.8581455945968628, -1.038008451461792, 1.9132086038589478, 1.5153164863586426, -0.45351195335388184, -0.792806088924408, -0.4689709544181824, -0.2878187596797943, 1.141640067100525, 0.2917101979255676, -0.3106071650981903, 0.1229255273938179, 0.19504575431346893, 1.322490930557251, 1.0730335712432861, -0.4404565989971161, 0.6447300314903259, -0.9240562915802002, 0.25035327672958374, -0.037261929363012314, 2.7081799507141113, 1.1078990697860718, 2.3129160404205322, -0.24900968372821808, -1.5487140417099, 1.0103610754013062, 0.0645921528339386, 0.9907137155532837, -1.5244009494781494, 1.1418153047561646, -0.19801916182041168, -0.6423813700675964, -1.3238580226898193, -1.4952964782714844, 0.8334622979164124, 0.04301366209983826, -1.8856956958770752, -0.22604763507843018, -0.8695349097251892, 0.2624037265777588, -0.14965416491031647, -1.694641351699829, 0.6267586946487427, -0.06129218637943268, -0.5636374950408936, -0.1828206181526184, -1.8361661434173584, -1.3754966259002686, -1.6337476968765259, 1.4939649105072021, 0.7731728553771973, -0.6184201240539551, 0.8070755004882812, -0.38345810770988464, -0.12600018084049225, 1.35151207447052, -1.1918023824691772, 0.7289537787437439, 0.7785239815711975, 1.8513281345367432, 0.6966608166694641, 0.05836072191596031, 0.9501195549964905, 0.715178906917572], [1.8456567525863647, 0.016121869906783104, -1.1457096338272095, -1.0926090478897095, 0.3676936626434326, -0.4531921148300171, 0.29820799827575684, -0.06494922935962677, 0.340469092130661, 0.8228362202644348, -0.7805496454238892, 0.5212719440460205, 1.3767622709274292, 0.22853142023086548, -1.2875219583511353, -0.10440532863140106, -0.48125481605529785, 0.24560441076755524, 0.9406117796897888, -1.9109033346176147, 0.6280605792999268, 0.22499297559261322, -0.39503413438796997, -0.1826472133398056, 0.45168882608413696, 0.2423200011253357, 0.40020403265953064, -0.4694185256958008, -0.01843148097395897, 0.8463267683982849, 2.4809396266937256, -1.9150843620300293, -0.030331552028656006, 2.4899215698242188, -0.41925081610679626, -0.10623792558908463, 0.28078320622444153, 0.7100130915641785, -0.9304776191711426, -0.684995174407959, 0.6414282917976379, 0.5158471465110779, 1.1161748170852661, 0.06431478261947632, 0.5130229592323303, 1.3776583671569824, -0.6909348964691162, -0.08932026475667953, 1.3142571449279785, 0.09382978081703186, -0.8129457235336304, -0.0925467386841774, 1.0358052253723145, 2.299546480178833, -1.1617014408111572, -0.17050454020500183, 0.27130380272865295, -0.3960391581058502, -0.7154041528701782, -0.7770976424217224, 1.0981905460357666, -1.7216883897781372, -1.0240333080291748, -0.11115157604217529], [0.023354891687631607, -2.4352056980133057, -0.44023236632347107, -1.0149774551391602, 1.0250385999679565, 0.5057535767555237, 1.5409168004989624, -1.4142918586730957, -0.3349836468696594, -0.7435126900672913, -1.5673398971557617, 0.5743982791900635, 0.3155865967273712, 0.6979535818099976, 0.5303337574005127, -1.0294455289840698, -0.11952097713947296, 1.5687980651855469, -0.2526804804801941, 0.8053881525993347, -1.9139400720596313, -0.4400791823863983, -0.086866095662117, -1.7047882080078125, -0.37354499101638794, 0.6625978350639343, -0.5599011182785034, 1.8125256299972534, -0.3392091393470764, 2.7462806701660156, 0.5892263650894165, -0.9255581498146057, 1.1029107570648193, 0.9007138013839722, -0.5873216986656189, -0.8315675258636475, -0.105500727891922, 0.9047946929931641, -0.5805870294570923, 0.534602165222168, -0.2180309295654297, 1.6314773559570312, -1.1723212003707886, -0.001346587436273694, -0.4283328354358673, -1.344510555267334, 0.15736988186836243, -0.007584200706332922, -0.2796449363231659, -1.091031789779663, -0.2708872854709625, -0.10586915910243988, 0.6824833154678345, 0.4238685667514801, -0.14042861759662628, -0.16104745864868164, 0.360460102558136, 0.3464626967906952, -1.2568923234939575, 0.18125873804092407, -1.2577344179153442, 0.8492183685302734, -0.07414520531892776, 0.7105056643486023], [0.3575838804244995, -1.7143303155899048, -1.6084251403808594, 0.5045011043548584, -0.8419211506843567, -0.12737704813480377, -1.1115185022354126, 0.2781165838241577, 0.13890340924263, 0.5205839276313782, -0.6245394349098206, 0.11739755421876907, 0.3167040944099426, 1.4959830045700073, 0.7563810348510742, 0.3725697100162506, -1.4862078428268433, 0.2981138229370117, 1.8706873655319214, -0.34142035245895386, -0.47261157631874084, -0.24265098571777344, 0.6347633004188538, 0.2843239903450012, -1.2609500885009766, 0.3269892632961273, 1.4603228569030762, -0.4486934244632721, -0.19994814693927765, -0.8472690582275391, 0.1537187248468399, 1.2845089435577393, 2.2996346950531006, 1.2756931781768799, -1.0763847827911377, 0.0734899714589119, 0.8112454414367676, -0.6032300591468811, 0.2900243401527405, -0.38802674412727356, 0.450399786233902, 1.922105073928833, -0.21402345597743988, -0.7468047738075256, -1.3257575035095215, 1.0871009826660156, 0.04085890203714371, 1.2484184503555298, -0.4617706537246704, 0.14562267065048218, -0.5315483808517456, 0.46115559339523315, -2.076409101486206, -0.21085727214813232, -0.03095085173845291, 0.39283114671707153, 0.38284680247306824, 1.642714023590088, 0.2637353539466858, 0.05770524591207504, 0.5219005942344666, -0.9161127209663391, -0.14044047892093658, 0.26756003499031067], [1.019171953201294, -0.4758313000202179, -0.23128820955753326, 0.34292563796043396, -1.185389518737793, 0.9609580039978027, 1.4519585371017456, 0.9299609661102295, 0.15640859305858612, -0.17017467319965363, -0.32789909839630127, 0.05201401188969612, -0.6887969970703125, -2.101233720779419, 2.2860186100006104, -0.3729953169822693, -0.10995618253946304, -0.6305603384971619, 0.7986187934875488, -0.24064777791500092, -0.10956860333681107, -1.0858830213546753, -0.06908601522445679, -0.8635514974594116, -0.5921309590339661, -0.4220134913921356, 0.2585409879684448, -0.8369383215904236, 1.0710703134536743, -0.27597153186798096, 0.12571822106838226, 1.344637393951416, 0.5756083726882935, 0.15987974405288696, 0.8730211853981018, -0.07328962534666061, 1.398842692375183, 0.30817028880119324, -0.6091810464859009, 0.8261210918426514, 0.6815259456634521, 0.6144068837165833, 0.40520572662353516, 0.8444995880126953, 1.3075358867645264, -0.4312974512577057, -1.0995678901672363, -0.6969815492630005, 0.7744683027267456, 0.8535147309303284, -0.5501101016998291, -0.5770363211631775, 1.0942286252975464, -2.1642606258392334, 0.93804931640625, -0.3206043243408203, 1.9813774824142456, -0.28739166259765625, -0.5332551002502441, -0.9712166786193848, 0.5494028925895691, -1.3132579326629639, 0.3392582833766937, 1.1710989475250244], [0.35791483521461487, 1.1644595861434937, 0.159520223736763, -0.8929253220558167, 0.7647914290428162, -1.1751822233200073, -0.6110923290252686, -0.7449436187744141, 0.32679784297943115, -2.065641403198242, 0.24822913110256195, 1.6467218399047852, -1.3795799016952515, 0.9241987466812134, -0.678555965423584, 0.8674036860466003, 1.178466796875, -1.3596941232681274, 0.6625475883483887, 0.35348275303840637, 1.6486722230911255, 0.8974975347518921, -0.029945755377411842, 1.20931875705719, -0.1718655824661255, -0.23393277823925018, -0.707444965839386, -1.5155164003372192, 0.7319514155387878, 0.7272598743438721, -0.9629551768302917, -0.7247051000595093, 1.2459988594055176, -0.03285969793796539, 0.8272818326950073, -1.7166939973831177, 0.08349527418613434, 0.25133436918258667, -0.4312392473220825, 1.7259169816970825, -2.730010986328125, -0.3442476689815521, -0.9251765012741089, -1.900031328201294, -1.0158146619796753, -0.4703875184059143, -0.7803165912628174, -0.9345869421958923, -1.6243667602539062, 0.5449103116989136, 0.7710571885108948, 0.4658425748348236, -1.8614439964294434, 1.4051955938339233, -0.8327519297599792, -0.5800533890724182, 0.3124086558818817, 0.16713419556617737, 1.3079383373260498, -0.5129084587097168, -0.19031944870948792, -0.12169719487428665, 1.422835350036621, 0.9103736281394958], [0.08092470467090607, -0.6765077114105225, 1.6103065013885498, 0.42515718936920166, 1.1343077421188354, -1.015641689300537, -0.36528629064559937, 0.2893454134464264, 0.408937007188797, 0.5138115882873535, -0.5690943598747253, 0.7958714962005615, -1.3540879487991333, -1.2402015924453735, -1.212012529373169, -1.9780434370040894, 1.0165616273880005, -2.221599817276001, 0.06654916703701019, 0.14039379358291626, 0.1742103546857834, 1.7849531173706055, 0.46324101090431213, -0.6269814372062683, -0.7260503768920898, 1.3166472911834717, -1.1488878726959229, 2.1133503913879395, 0.09004198014736176, -1.0597420930862427, 0.14904966950416565, 1.0689560174942017, 0.5779018998146057, -0.23095154762268066, 0.12551937997341156, 0.3746986985206604, 2.4718234539031982, -0.03590559959411621, 0.4100237190723419, -1.3090897798538208, -0.6332602500915527, 0.2984216809272766, -1.232849359512329, 0.2640092372894287, -0.10549797862768173, -0.6066597700119019, 0.7871108055114746, -0.8468414545059204, -0.16014143824577332, -0.5390767455101013, -1.1621638536453247, 0.6073352694511414, 1.587831735610962, -0.42704349756240845, 0.05387209355831146, 1.013165831565857, -1.1384645700454712, -1.264181137084961, -0.2938059866428375, -2.343088150024414, -0.13473308086395264, 1.3164643049240112, -0.5150443911552429, -2.826089859008789], [1.4518455266952515, -0.4398922026157379, 0.519547700881958, 0.43595489859580994, 0.8546277284622192, -0.8278224468231201, -1.3732236623764038, -0.2892810106277466, 0.19957134127616882, 2.1174330711364746, -0.0381670743227005, 0.19019633531570435, 0.9685176610946655, 2.1543524265289307, -0.17147204279899597, -1.4468615055084229, -0.9807051420211792, -0.6572442650794983, 0.6121289134025574, 0.1364697962999344, -0.3324148952960968, 1.7641624212265015, 0.014243829064071178, -0.4422120153903961, -0.6689488887786865, 0.06223195418715477, 1.2173603773117065, -0.633549153804779, 0.24470804631710052, -0.19193749129772186, 0.10944759100675583, 1.1822398900985718, -1.2949022054672241, 0.931460976600647, 0.4307037591934204, 1.247070074081421, -0.6933240294456482, 0.6698899269104004, 1.5495502948760986, 0.8290548324584961, -0.8558121919631958, -1.0540976524353027, -0.1806766837835312, -0.6414518356323242, -0.0794399157166481, -0.391200989484787, -0.4206821918487549, -0.5085523128509521, 0.435770183801651, -0.20058605074882507, -0.3507719933986664, -0.3551040589809418, 0.004082075320184231, 0.34415918588638306, -1.201354742050171, 1.1704212427139282, 0.10438370704650879, 0.22965116798877716, 0.14629791676998138, 0.39160019159317017, 2.0186047554016113, -1.1806834936141968, -2.036674976348877, -2.0989975929260254], [-0.20703651010990143, -1.1863702535629272, 0.7853657603263855, 1.468088150024414, 0.8245751857757568, -2.3573710918426514, -0.3143743872642517, -0.7548598051071167, -0.5227259993553162, 0.973834753036499, 0.8390758633613586, 0.8362895250320435, 0.23461434245109558, -1.0976994037628174, -0.037450697273015976, 0.43042147159576416, 0.47732529044151306, -2.1248321533203125, 0.3114669919013977, -0.8003429174423218, 0.3733201026916504, -0.1553843468427658, 0.4043072760105133, -0.45218679308891296, 2.318182945251465, -1.4229439496994019, -0.7284722924232483, -0.5382965207099915, -0.8022446036338806, 0.006462722551077604, -1.0642341375350952, -1.8249199390411377, 1.2428737878799438, 1.3250086307525635, -0.5358231663703918, 1.7288398742675781, -0.7238079905509949, -0.026619529351592064, 1.4811781644821167, -0.42112910747528076, -0.5712236166000366, 0.7898827791213989, 0.009070371277630329, 0.9267063736915588, -0.9129164814949036, -0.09335047006607056, 1.1932308673858643, 0.15623629093170166, -0.15996462106704712, 1.2123974561691284, 0.6600155830383301, -0.8910540342330933, 0.6398581266403198, -1.6550853252410889, 0.17207592725753784, 0.13366356492042542, 2.3719310760498047, 0.30130213499069214, -1.6399269104003906, 0.7396772503852844, 1.0380678176879883, -0.6577882170677185, -1.444549798965454, -1.374409794807434], [1.2550456523895264, -0.13149502873420715, 1.182820439338684, -1.617809534072876, -0.6716060042381287, 0.6175042390823364, 1.7202427387237549, -0.9700908064842224, 1.5684802532196045, -0.44752931594848633, 0.008351432159543037, -0.16979283094406128, 0.2838863730430603, 0.527142345905304, 0.13053947687149048, -1.1954697370529175, -1.2821054458618164, 1.0094646215438843, 0.5368393659591675, -0.7662866115570068, 0.19103483855724335, -0.7409334182739258, 0.33129173517227173, -0.43592947721481323, -0.5946769118309021, 0.6051437854766846, 1.2226632833480835, 0.8234069347381592, 0.9046323299407959, -0.557604193687439, 0.57138991355896, -0.421307772397995, -0.8928394317626953, 0.09383466839790344, -0.39969736337661743, 1.1693651676177979, -0.1726303994655609, -1.2969300746917725, 1.3895139694213867, 0.1130441203713417, 2.7533199787139893, -0.47282832860946655, 0.05774815008044243, 0.030678778886795044, -1.258568525314331, -0.4665043354034424, 0.3586587905883789, 1.0659315586090088, -0.04707220196723938, -1.515401005744934, -1.2025907039642334, 0.5132694840431213, -0.24015827476978302, 0.8296886086463928, 0.8875692486763, 0.5532475113868713, -0.1220141351222992, -0.42133256793022156, -0.19765238463878632, -0.1110522449016571, -1.0633234977722168, -0.5628191828727722, -0.04475049674510956, -0.691635012626648], [-0.5001603960990906, 1.70132315158844, -0.5872864127159119, 0.11337627470493317, -0.20733891427516937, 0.41946372389793396, -0.38355278968811035, 1.608579158782959, 0.09554792940616608, -0.36238712072372437, 0.5052503943443298, -1.209503173828125, -0.29370754957199097, -0.3240128457546234, -0.3490596413612366, 1.086163878440857, -0.599331259727478, -1.791881799697876, 0.9260090589523315, 0.6861588358879089, -0.4540255069732666, 1.0492620468139648, 0.5443190932273865, -1.1674065589904785, -0.5410979986190796, -1.1392923593521118, -0.37154126167297363, 0.6678487658500671, 0.05130137875676155, -0.38688117265701294, 0.8033251166343689, 1.482746958732605, -1.0310711860656738, -0.6476057171821594, -0.3728802502155304, -0.18688571453094482, 0.30718502402305603, 1.94319748878479, 1.3330886363983154, -0.7076255083084106, 0.4300941228866577, 0.34428369998931885, -0.8700418472290039, 1.2727713584899902, -0.880095362663269, -0.6074638962745667, -1.2358750104904175, -1.194421648979187, -0.1768697202205658, -0.39302054047584534, -0.11399626731872559, -1.4008089303970337, -1.1765661239624023, -0.1679982990026474, -0.9050934910774231, 0.3103184401988983, -0.912968635559082, 0.8286533355712891, -0.2568435072898865, -0.6581811904907227, 0.6490001082420349, -0.6078426837921143, 1.849439024925232, 0.9196824431419373], [0.2800377607345581, -0.4362495243549347, -0.3623521625995636, -0.013734386302530766, -0.3986130356788635, 1.143349528312683, 0.5825995802879333, 0.8453340530395508, 0.06062278524041176, -1.8095461130142212, -1.9823721647262573, -0.6875381469726562, 0.4571736752986908, 2.167459726333618, -2.0939950942993164, -1.1901493072509766, -0.23580650985240936, 0.9485352039337158, 0.7028096318244934, -1.28132164478302, -0.35784751176834106, 1.18455970287323, -1.1120721101760864, -1.3640602827072144, 3.5404434204101562, 0.7749577164649963, -1.6076444387435913, 1.5528801679611206, -0.752001166343689, 0.27897581458091736, 0.6491473913192749, -1.5149562358856201, -0.06353040039539337, 0.39174872636795044, 0.19342397153377533, 1.3332479000091553, 0.7870336771011353, 0.9504596590995789, 0.943406879901886, -1.5605552196502686, -1.0337632894515991, -1.5071882009506226, 0.1538943201303482, -0.6574468016624451, -0.6679245829582214, 1.3489141464233398, -0.9589730501174927, -0.745437502861023, 0.4478384852409363, 0.8729004263877869, 0.4013071060180664, 0.4892931580543518, -1.2059074640274048, -0.5921456217765808, -0.1621265411376953, -0.6664156913757324, -1.98952317237854, -1.1528255939483643, -1.2520009279251099, -0.18390579521656036, 2.0178449153900146, -1.04221773147583, 0.43917983770370483, -0.3020004630088806], [0.631640613079071, 0.6005714535713196, 0.7808706164360046, 0.624030590057373, -0.2975887954235077, -2.6394753456115723, 2.0681138038635254, 0.11670301854610443, -2.090907335281372, -1.4262826442718506, 0.21030794084072113, -0.8479019403457642, 0.2592794895172119, -1.5784043073654175, 1.4431428909301758, -0.28030434250831604, 1.6555697917938232, -0.13907210528850555, 0.09221865236759186, -0.10417202860116959, -0.34358111023902893, -0.8264185786247253, -0.4100572168827057, 0.040676653385162354, 0.631239652633667, -0.8193107843399048, -0.47824227809906006, -1.027016282081604, -0.12100983411073685, -0.1705683469772339, 0.016848653554916382, -1.093778133392334, -0.863794207572937, -1.4635977745056152, 1.1610639095306396, -0.0950293242931366, 1.109253168106079, -0.5040469169616699, 1.1007431745529175, -1.2940232753753662, 2.8384861946105957, 0.3028252422809601, -0.40278199315071106, -0.09858354926109314, -0.10566821694374084, 0.02845575287938118, -0.22609516978263855, 1.139045000076294, -1.6936852931976318, 0.960867702960968, -2.904815912246704, -0.6226367354393005, 0.37978219985961914, -0.2081497460603714, -0.6543052196502686, 0.7644122838973999, -0.6304136514663696, -1.5747300386428833, -0.6214290261268616, -2.180182695388794, -1.4351775646209717, -0.5556175708770752, 0.35439008474349976, 1.4605432748794556], [-1.4659309387207031, 0.849345326423645, -1.5501357316970825, 0.47083476185798645, 0.7074496150016785, -0.2732948064804077, 0.5487103462219238, 0.06141512095928192, 0.3864818513393402, 0.21661438047885895, 1.3586721420288086, -1.53300142288208, -1.0564031600952148, -0.22274067997932434, -0.12551695108413696, -1.3975101709365845, 1.213802695274353, -0.503950297832489, -0.4566315710544586, 1.1402424573898315, 0.3066296875476837, 1.6778085231781006, 1.4436976909637451, -0.6522461175918579, -2.4814350605010986, -1.7425215244293213, -2.245323657989502, 0.8206751346588135, -1.4050476551055908, 0.5654134750366211, 1.0473573207855225, -1.517187476158142, 0.7574543952941895, 0.6263757348060608, -1.6268723011016846, 1.6149348020553589, 0.7910513877868652, 1.2014135122299194, -0.8628196716308594, -2.917450189590454, 0.5534494519233704, 0.5036609768867493, 1.4636163711547852, -1.6206169128417969, -1.3916044235229492, -0.7682653665542603, -0.24619005620479584, 0.20842154324054718, 1.0149136781692505, -0.9079263806343079, -0.741875410079956, 1.2776904106140137, 0.44587257504463196, -0.5701159834861755, -0.8556622862815857, -1.9791464805603027, -0.27754539251327515, 1.1736278533935547, -0.05754857137799263, -0.8337163329124451, 0.3052634596824646, -0.10435664653778076, 0.08475039899349213, -0.5589152574539185], [-0.6955219507217407, -0.6837811470031738, -0.39208653569221497, 0.8890829086303711, -1.171920657157898, -0.10985175520181656, -0.46728309988975525, -1.5077224969863892, 0.5165122747421265, 1.388953685760498, -1.0699957609176636, 1.3462368249893188, 2.810528039932251, -0.7596078515052795, 0.5950984954833984, 0.6810584664344788, 0.3050733804702759, -0.8768683671951294, -0.0379272922873497, -2.247896909713745, -0.31276699900627136, 1.894343376159668, 1.6861541271209717, -0.10189563035964966, 0.8378101587295532, 0.8654901385307312, 0.6207211017608643, -2.41306471824646, -0.055233508348464966, -0.015125657431781292, -0.16255028545856476, -0.8071427941322327, -1.1605130434036255, 0.4996338188648224, -2.2760298252105713, 2.722867727279663, 0.3319745361804962, -0.8782635927200317, -0.5515177845954895, 2.0141773223876953, 1.4060825109481812, -1.278549075126648, 1.0810370445251465, 0.5968435406684875, 0.5513713359832764, -0.2512175142765045, -0.7206990718841553, -1.044230580329895, 0.4534512758255005, -1.316430926322937, 0.46192896366119385, -1.2745598554611206, -0.6529524326324463, -0.12546087801456451, -1.0731003284454346, -0.33528873324394226, -1.5430530309677124, 0.8945099711418152, -0.7842419147491455, -0.8650234937667847, -0.28700950741767883, -0.20007047057151794, -1.4634113311767578, -2.0826926231384277], [1.74111008644104, 0.10787840932607651, -0.29797104001045227, -0.18129193782806396, 0.050652556121349335, 0.06799663603305817, -0.34807366132736206, 0.31195592880249023, -0.06315169483423233, -0.3094332814216614, -0.12388159334659576, -0.5739360451698303, -0.1863509863615036, -0.1654658317565918, -0.4782777726650238, -0.5255431532859802, -0.8274348378181458, 0.28325459361076355, 0.2959565818309784, -1.7066380977630615, 0.5359017848968506, 0.019793109968304634, 0.9802320599555969, 0.5844690203666687, -0.4754013121128082, -0.8119696378707886, 0.6263564825057983, 0.6039423942565918, 0.2408263087272644, -0.9637539982795715, -0.5406590700149536, 1.1954532861709595, -0.9220014810562134, 1.4278194904327393, -0.5404437780380249, 1.142402172088623, -0.9536662697792053, 0.10350993275642395, 0.4238855242729187, 0.22447451949119568, 0.24489548802375793, -0.5547422170639038, -0.5609860420227051, -0.026482366025447845, 1.5135189294815063, -0.9848496913909912, -1.3886445760726929, 1.5507261753082275, -0.42814671993255615, -1.5819982290267944, -0.7663577198982239, -0.42995575070381165, 0.846764326095581, -0.3850283920764923, -0.06557714194059372, 0.042122144252061844, 0.30222928524017334, -0.536563515663147, -0.07649553567171097, 1.0976550579071045, 0.140662282705307, -0.32176244258880615, 0.4639296233654022, 1.1678242683410645], [-1.197596788406372, -0.9344943761825562, 0.17525525391101837, -0.1195070743560791, 0.6402186155319214, 0.5665686130523682, -0.8991826772689819, -1.1010359525680542, 0.5061873197555542, -0.007759207859635353, 1.762189507484436, 0.5577254891395569, 0.7400176525115967, 0.18344730138778687, 0.7547808885574341, -0.7767572402954102, -1.225143551826477, -1.9071606397628784, 1.1217564344406128, -1.3862552642822266, 0.41879984736442566, 1.0496020317077637, -0.6532975435256958, -0.15544404089450836, -0.46998828649520874, 0.20298950374126434, -1.0652531385421753, 0.984055757522583, 1.245186448097229, -0.9567845463752747, 1.1519078016281128, -0.29721492528915405, -1.3789194822311401, 1.2101409435272217, 0.10360071063041687, -0.5815753936767578, -0.24873530864715576, -1.623586654663086, -1.3698049783706665, -0.9208191633224487, -1.262491226196289, 0.03913475573062897, 0.39937281608581543, -0.6744238138198853, -1.0997581481933594, -0.3309139013290405, -0.3784804940223694, 0.5095363855361938, 0.8700671792030334, 0.8790981769561768, 0.8208461999893188, 1.6260197162628174, 1.1085020303726196, -0.0019178008660674095, 0.35002079606056213, -1.5297292470932007, -0.8378369212150574, -1.0729807615280151, 1.7646688222885132, -0.936060905456543, -0.9348481297492981, -0.2714061141014099, -0.44531184434890747, 1.855593204498291], [-0.6933145523071289, -0.7151782512664795, 0.3157651126384735, -0.8652264475822449, -1.0153666734695435, 0.632572591304779, 0.31017327308654785, -1.0520002841949463, 1.72962486743927, -0.0631217435002327, 0.1791245937347412, 0.051499463617801666, 0.124811090528965, -0.9674782752990723, 0.12574927508831024, -0.9486660957336426, -0.1694556325674057, 0.5618219971656799, 1.2182555198669434, 0.3675917088985443, 0.11271923035383224, -0.8790358901023865, -1.3511155843734741, -1.4438618421554565, -0.8338513970375061, 0.7504048347473145, 2.8895416259765625, -0.35267287492752075, -0.4767294228076935, -1.5715601444244385, -0.3383864164352417, 0.21767382323741913, -0.7185524702072144, 0.2434980273246765, 1.1389102935791016, 0.6834911704063416, -1.1177200078964233, -0.2128540277481079, 1.5122548341751099, 0.9029571413993835, -1.1311715841293335, 0.2639963626861572, 2.010099411010742, -0.8449628949165344, 1.4696966409683228, 0.6864884495735168, -0.039139531552791595, 1.3896803855895996, -2.0589282512664795, 0.9666184782981873, 0.10971920192241669, 1.0869051218032837, 0.23999209702014923, -0.12614336609840393, -0.9475660920143127, 0.29205846786499023, -0.5862661004066467, 1.8420416116714478, -0.9663981199264526, 0.21693015098571777, 1.4515520334243774, -0.918091356754303, 1.9285409450531006, -1.5551015138626099], [-1.9990254640579224, 1.0391335487365723, 0.5196353197097778, -0.8928717374801636, 1.3844789266586304, 1.2679935693740845, 1.3822118043899536, -0.2925238609313965, -1.612208604812622, 0.1697433441877365, -1.03673255443573, -1.63432776927948, -0.6291801929473877, 1.1078202724456787, -1.602695107460022, 1.479082465171814, 1.4693090915679932, -1.2446563243865967, 1.983981966972351, 0.06934165954589844, 0.8257231712341309, 0.2773837447166443, -0.4147247076034546, 0.20658992230892181, 0.29817312955856323, 0.10364428907632828, 0.5810602903366089, -0.9701559543609619, -1.0874682664871216, 0.7330408096313477, -1.9763307571411133, -1.7959109544754028, 0.8679218292236328, -0.5421361923217773, 0.7352185249328613, 1.2384957075119019, 1.7090548276901245, -0.326200008392334, 0.5232509970664978, 0.5300057530403137, 0.7532649636268616, 0.14651474356651306, 0.24605338275432587, 1.6741909980773926, 0.6267330646514893, 1.6421384811401367, 0.16835470497608185, 0.39173489809036255, 0.8253015279769897, 0.16350220143795013, 0.602780818939209, 1.6770535707473755, -0.18477292358875275, -1.6258841753005981, -0.29468169808387756, 1.3286941051483154, 0.38501232862472534, -0.1907723844051361, -0.5837894082069397, -0.41301485896110535, -1.7920349836349487, -1.7636587619781494, -0.9946863651275635, 0.47647878527641296], [0.9230843186378479, -0.6807739734649658, 0.6458885669708252, -1.478184461593628, 0.14452090859413147, 0.812736451625824, -0.14560258388519287, 0.7425750494003296, 0.41989612579345703, -0.6129119992256165, 1.2973055839538574, -0.2738012671470642, 1.4152470827102661, 0.03956207260489464, 0.6299857497215271, 2.0394790172576904, 0.3575413227081299, -0.6551377773284912, -0.34218186140060425, -0.9979918599128723, -0.6941250562667847, -1.4967652559280396, -0.49769511818885803, -0.7357943654060364, -0.5734648704528809, 0.8675866723060608, -0.5611623525619507, 0.3244750201702118, -0.6438825726509094, -1.113431453704834, -0.7385719418525696, 1.2680021524429321, -0.28261515498161316, -0.3538382947444916, -0.04311707243323326, 0.28425008058547974, -0.9856863617897034, 1.78067147731781, 1.6005477905273438, -1.4619200229644775, -1.415270209312439, 0.29757633805274963, -1.0028035640716553, 1.668161392211914, 0.7460717558860779, 1.9326856136322021, 1.2538763284683228, -1.5664862394332886, 1.5285574197769165, -0.16198424994945526, 1.652919888496399, 0.5786056518554688, 0.21520276367664337, 2.060581922531128, 0.6718835830688477, -0.9063175916671753, 0.3468054234981537, 1.8379435539245605, -0.9187139272689819, 0.8350799083709717, -1.2271525859832764, 0.5526974201202393, 1.6034860610961914, 0.10253431648015976], [-0.16541622579097748, 0.5801171660423279, -1.8867769241333008, -0.6626901626586914, -0.6510966420173645, 2.0485024452209473, -0.4432887136936188, -0.2389782816171646, -1.0246237516403198, 0.5163654685020447, 0.342362642288208, -0.18712259829044342, -2.5418827533721924, 1.3188276290893555, -0.596807062625885, -0.7422792911529541, -0.6582595109939575, 1.7257200479507446, -0.04750312864780426, 2.1487176418304443, 0.021565550938248634, 0.8215084075927734, 0.9829520583152771, -2.387289047241211, -0.6941781640052795, 0.9667525291442871, 1.2752991914749146, 0.9406821727752686, 0.19562333822250366, -0.5492246150970459, -2.6356091499328613, 0.8549259305000305, 0.5722635984420776, 0.02034597098827362, -0.06502224504947662, -0.010088274255394936, 0.7985355257987976, 0.06015109270811081, -1.0610638856887817, 0.007494671735912561, 1.2459068298339844, 0.12206259369850159, 0.37851065397262573, 0.1789221465587616, -0.5258963704109192, -0.3599534034729004, 0.13196294009685516, -1.6803174018859863, -0.13146044313907623, -1.0933690071105957, -0.29040172696113586, -1.0490734577178955, -1.215807318687439, 0.6202518343925476, 1.8542959690093994, -1.0316309928894043, 1.9498027563095093, -0.8037999272346497, 0.9643037915229797, -0.8347324728965759, -0.7553110718727112, -0.4515495300292969, -0.036178868263959885, 2.974973440170288], [0.058474961668252945, 1.2754662036895752, 0.9178571701049805, -0.669430136680603, 1.005139946937561, 0.13498513400554657, -0.3201867938041687, 0.2236630767583847, -1.5555812120437622, 0.24039748311042786, 0.28832265734672546, -0.7202474474906921, -0.6090127825737, 0.18489785492420197, -1.1471538543701172, -0.6986793279647827, -1.6050288677215576, 0.7906568646430969, 1.10124671459198, 1.099034309387207, 0.5297998189926147, 0.9511558413505554, -0.49973756074905396, -0.15961262583732605, -0.5978776216506958, -0.8545584082603455, -1.813578724861145, 0.9757828116416931, 0.5972091555595398, 0.44345220923423767, -0.38200142979621887, 0.4105759263038635, -0.8740207552909851, -0.4559210240840912, -1.27996027469635, -1.065569281578064, -1.0837668180465698, -0.7119253277778625, 0.1565873920917511, -0.08267740905284882, -0.6117263436317444, 0.06021489202976227, 1.0619065761566162, 0.47301116585731506, 0.5110512375831604, 1.345254898071289, 0.3018311560153961, 1.9961751699447632, -1.2727755308151245, -0.5271884202957153, 0.4693020284175873, 1.558547019958496, 0.6199869513511658, -1.6458699703216553, 1.474633812904358, -0.6001805663108826, -0.7786754965782166, -0.8606253266334534, 1.054783582687378, 0.7502321004867554, 0.14726383984088898, -0.8821210265159607, 0.3773330748081207, -2.6800782680511475], [-1.4994254112243652, 0.8237424492835999, 0.33531221747398376, 0.48294803500175476, -0.1720961034297943, 0.8350422382354736, -0.0856810063123703, 1.5129458904266357, 0.5312179923057556, 0.4732702672481537, -1.029431939125061, -0.3284737169742584, 0.5997710227966309, 0.40038612484931946, 0.6105139255523682, -0.15993019938468933, -1.3489798307418823, 0.879767894744873, -0.5137080550193787, 0.22221435606479645, -1.093026876449585, 0.7725842595100403, -1.1454259157180786, 0.44075077772140503, -0.41256576776504517, -0.5535182356834412, 0.51053786277771, 0.2732780873775482, 0.5502453446388245, 0.6745683550834656, 1.492695689201355, 1.2219692468643188, -0.38326990604400635, -0.16492412984371185, -1.464845895767212, -1.209898829460144, -1.0001329183578491, -0.20681734383106232, 0.5836294293403625, -0.4623742699623108, 0.9630516171455383, 0.6960658431053162, -0.06107735261321068, -0.8443366885185242, -1.1752004623413086, -0.37186911702156067, -1.492291808128357, 0.42217379808425903, -0.19612251222133636, -0.9875746369361877, 0.5556467771530151, 0.021140623837709427, -0.6579927206039429, 0.39723506569862366, -0.1388511210680008, 2.363351345062256, -0.7313905358314514, 0.9731977581977844, 1.5872002840042114, 0.30619367957115173, 0.013925513252615929, -1.41660475730896, -1.0038808584213257, 1.228409767150879], [0.2231515347957611, 0.814924955368042, -0.11597242206335068, 0.36904752254486084, -0.3364405035972595, -0.15272745490074158, 1.2190883159637451, 0.007313739974051714, -0.5853687524795532, -0.240953266620636, -0.018452348187565804, 0.7178175449371338, -0.5068697333335876, -0.41155946254730225, 0.6018995046615601, -0.9822801947593689, 0.499657541513443, 1.0881779193878174, 0.670563280582428, -2.695697784423828, 1.0974634885787964, -1.322608470916748, 1.3335847854614258, -0.009039320982992649, -0.03915425390005112, -1.1384655237197876, 1.2768278121948242, 0.18165916204452515, 0.36866825819015503, 0.10858746618032455, -0.15889397263526917, -1.2471667528152466, 0.7596448659896851, 0.1929950714111328, -1.4621806144714355, 0.9078683853149414, -0.03213436156511307, -0.7791755795478821, 1.2812864780426025, 0.8206570744514465, -0.9253790974617004, -1.0722509622573853, 0.9119800925254822, 1.3578189611434937, -0.6836484670639038, 1.134814739227295, -0.9125549793243408, 0.5245670080184937, 0.9606685638427734, -0.9634698033332825, 0.09667299687862396, -1.3722753524780273, -2.3416786193847656, -0.14371713995933533, 0.6626132130622864, -0.5042562484741211, 0.19123798608779907, -0.09330004453659058, 1.1340869665145874, 0.27396678924560547, -0.32526183128356934, -1.5923610925674438, -0.2439650595188141, -1.0477485656738281], [1.416296362876892, -0.12508004903793335, -0.12523595988750458, 0.1702287495136261, 0.14749319851398468, 0.6582943201065063, 0.33279573917388916, -0.6844731569290161, 2.6865158081054688, 0.27526023983955383, 0.18690964579582214, 0.33614709973335266, -0.5700348019599915, -0.12759359180927277, 1.4806098937988281, -1.523491621017456, -0.653019905090332, -0.4783347249031067, 1.8346458673477173, -1.3068172931671143, -0.08569808304309845, -0.22547177970409393, -0.4750918745994568, -1.2203766107559204, -1.6905277967453003, -2.0992562770843506, 0.2718682885169983, -0.4234435260295868, 0.5614701509475708, -0.9859082102775574, 1.5994389057159424, 0.7641545534133911, -3.184725046157837, 1.1013355255126953, -2.8932557106018066, 3.02842378616333, 1.211961269378662, 1.0896540880203247, -0.13605526089668274, -0.22996219992637634, -1.28162682056427, -0.9097364544868469, -1.403823733329773, -0.9681737422943115, -0.30411800742149353, 1.3666231632232666, -0.5784085392951965, -0.6262921094894409, -0.3118516504764557, 0.31739431619644165, -0.40814366936683655, -0.5703204870223999, 0.860467791557312, -0.32104557752609253, -1.1388007402420044, -0.9631548523902893, 0.029491188004612923, 0.5496131181716919, -1.9246127605438232, 0.5652173757553101, 1.1930115222930908, -0.7682347893714905, -0.964905321598053, -0.6866733431816101], [-1.3795274496078491, 0.12538118660449982, -1.3091388940811157, -1.3416582345962524, 1.7662941217422485, -0.6637105941772461, -1.4732517004013062, 1.3537887334823608, -0.0022673013154417276, -0.38187897205352783, 0.014400500804185867, -0.8652997612953186, 1.4986597299575806, 1.6376649141311646, 0.8061647415161133, 0.7840176820755005, 1.9042234420776367, 0.31984245777130127, -0.19532933831214905, 1.591149926185608, 1.8907668590545654, 1.4796652793884277, 0.19718675315380096, 1.0309810638427734, 0.5948862433433533, 1.370444416999817, 1.477025032043457, 0.40241539478302, 1.1603561639785767, 1.6039332151412964, -0.17544254660606384, 0.8041833639144897, -0.24752023816108704, 1.5928493738174438, -1.7007899284362793, -0.5631648898124695, -1.5115019083023071, 1.257577657699585, 0.3498319387435913, 1.04445481300354, -0.7536541819572449, 0.1698777824640274, 0.8819093108177185, -0.25948449969291687, -1.2739381790161133, -0.11352456361055374, 1.2202680110931396, 0.15203560888767242, 0.8031566739082336, -0.6496610045433044, 0.16575537621974945, -0.4100272059440613, -0.4605710804462433, -0.5134534239768982, 0.3014560043811798, -0.7384820580482483, -0.9620972275733948, -2.424697160720825, -0.316578209400177, -0.5412593483924866, 0.7090219855308533, -0.576065182685852, -0.19771194458007812, -0.14375965297222137], [-0.2781580090522766, 0.02616395615041256, 0.9763438105583191, -1.1695579290390015, 0.9840257167816162, -1.9939868450164795, -0.49237939715385437, 0.1436161994934082, 0.6592894196510315, 0.18885847926139832, 0.19327802956104279, -0.5500667095184326, 0.6908457279205322, 0.17732614278793335, -1.5614246129989624, -0.8748302459716797, -0.09099610894918442, -0.40812164545059204, -0.9959920644760132, 0.5486064553260803, 0.23979811370372772, 1.4139430522918701, 0.9298304319381714, 0.21450519561767578, 2.199552059173584, 0.261122465133667, -0.3890107274055481, 0.5780352354049683, -0.5437083840370178, -0.19627518951892853, -0.645384669303894, -0.04939989000558853, 1.4348526000976562, -0.722954511642456, 0.5254063606262207, 0.6855583190917969, -1.1778669357299805, -0.8341506123542786, -0.24144600331783295, 1.5986804962158203, -1.0727283954620361, 0.24328944087028503, -0.7643257975578308, 0.15376204252243042, 1.609291911125183, -0.4766254425048828, 0.8961654305458069, 0.23897100985050201, -0.7059942483901978, -0.4274565875530243, -0.5217239856719971, 1.1177103519439697, 0.8123795390129089, -0.8614410758018494, -1.175679087638855, -1.503704309463501, 1.0230034589767456, 0.29455623030662537, -0.8217376470565796, -1.4554612636566162, -1.634256362915039, 1.4221446514129639, -0.32806751132011414, -1.1438157558441162], [-0.4606476128101349, -0.5608980059623718, 0.22930262982845306, 1.0899145603179932, 2.4823312759399414, 1.0447643995285034, -0.745140016078949, 0.4226195216178894, 0.36690136790275574, -1.0737266540527344, -0.21866796910762787, -1.0644110441207886, -0.4331391751766205, -0.27436569333076477, -0.6346451640129089, -1.5621541738510132, -0.31951573491096497, -1.2140456438064575, -1.306203842163086, -1.1314717531204224, -0.48473700881004333, -0.06020699068903923, -1.006618618965149, 0.04880090430378914, 1.9107869863510132, -0.8609354496002197, -1.242553472518921, 0.9955059289932251, 0.5301529169082642, 0.6605586409568787, -0.0385294035077095, 0.22302420437335968, 0.6889565587043762, -0.4385416805744171, -1.1248104572296143, -0.40451326966285706, -0.6770445704460144, 0.1319315880537033, -0.24519410729408264, -1.088883876800537, 0.7095107436180115, -0.6592684388160706, -1.620435118675232, 0.6892493963241577, -1.9848134517669678, -1.6408404111862183, -1.1417653560638428, 1.2465784549713135, -1.4069926738739014, 1.0105760097503662, -0.6415660977363586, 0.1898013949394226, -0.32143211364746094, -0.5615451335906982, -0.5464522838592529, 0.7986181974411011, 0.3292865455150604, -0.45209363102912903, -0.1553114354610443, -1.5287669897079468, 0.0432158038020134, 0.5957449674606323, 0.07101670652627945, 0.45785045623779297], [-1.3831614255905151, -1.8078993558883667, -1.1278654336929321, -1.6874819993972778, -1.8718167543411255, -1.0676372051239014, 0.3913107216358185, 1.55074143409729, 0.6073110699653625, 0.4329356253147125, 0.1088559553027153, -0.6509566903114319, 1.1663435697555542, -1.0023142099380493, 0.09342387318611145, -0.25678732991218567, 0.48753705620765686, 1.9399864673614502, -1.8808002471923828, 0.22593864798545837, 1.2107856273651123, -1.4976948499679565, -1.1383992433547974, 1.3824840784072876, -1.4167909622192383, 0.02655263990163803, -1.8213810920715332, -0.05902760103344917, 2.3353428840637207, -1.2203326225280762, -0.01782844215631485, 0.635772705078125, 0.7887690663337708, 0.529431939125061, -1.8071517944335938, 0.3668462634086609, -0.8347247242927551, 0.06917351484298706, -0.7783074975013733, 1.0681443214416504, 0.016797903925180435, -0.8678427934646606, 0.26620855927467346, 0.705533504486084, 0.9453616738319397, -0.47291436791419983, 1.0941675901412964, 0.798514723777771, -0.8998026251792908, -0.24638287723064423, -1.1393542289733887, -0.5417112708091736, 0.5105233788490295, -0.1475740522146225, -0.1575886607170105, -0.3312758505344391, 0.06721045821905136, -2.071119546890259, 2.122816801071167, 0.7367045283317566, -0.10194504261016846, -0.24306975305080414, -1.955651879310608, 0.27901846170425415], [-0.07275725156068802, 1.010457992553711, 0.551220715045929, -0.3685624301433563, -0.23261138796806335, -0.5807558298110962, -2.6333675384521484, -0.47808852791786194, 1.8107624053955078, 0.7336695790290833, 1.2222861051559448, 1.714280605316162, 0.02858506143093109, -0.5186918377876282, -0.027095580473542213, 0.0757322832942009, 0.673088788986206, 0.27324429154396057, -0.21517300605773926, 0.05489429458975792, 0.6407551765441895, 1.1308702230453491, -2.2319860458374023, -0.5646266341209412, 0.5631273984909058, 0.11311201751232147, -0.21419332921504974, 0.6900793313980103, -0.6948704123497009, -1.226820468902588, -1.878427267074585, 0.20104339718818665, 0.6365455389022827, 0.6920965313911438, 1.7399752140045166, 0.5415039658546448, -0.09079213440418243, 0.8634981513023376, 0.6295466423034668, 0.5860574245452881, -2.030592203140259, -0.44941893219947815, -1.006198763847351, 0.4609328806400299, -0.8555227518081665, 0.9519141316413879, -0.33801835775375366, 0.3075140118598938, 0.2435799092054367, 0.1432921588420868, -0.25179898738861084, 0.7075601816177368, -0.029737848788499832, 0.7952907085418701, 0.41416871547698975, -0.28041398525238037, 0.21412484347820282, 0.6200546026229858, 0.7897564172744751, 0.5164166688919067, 0.7087804675102234, 1.3902941942214966, 0.327292263507843, -0.38650915026664734], [-1.299236536026001, 1.0437449216842651, -0.18169733881950378, -1.9736751317977905, 0.2079066038131714, 1.455155611038208, -1.6776291131973267, 1.5759330987930298, -0.061023011803627014, -1.1128677129745483, -0.37010297179222107, 1.751896858215332, 0.030270038172602654, 0.5256307125091553, -0.3569726347923279, 0.9566027522087097, 0.11821214109659195, 1.6552250385284424, -0.15395420789718628, 0.3160322904586792, -0.27290230989456177, -0.45655500888824463, 0.6868611574172974, -0.586002767086029, -0.4141576886177063, 0.88417649269104, 0.8694546818733215, -0.32842859625816345, -0.26578289270401, 0.14409220218658447, 0.036794744431972504, -1.1693776845932007, 0.7189986705780029, -1.4894262552261353, 0.8599955439567566, 1.6413058042526245, 1.2929050922393799, -2.1745259761810303, 0.11040648818016052, 0.49755021929740906, -1.5821691751480103, -0.7484709620475769, -0.18356789648532867, 0.8252123594284058, -0.10214024037122726, 0.30992281436920166, 0.02548414282500744, -2.512741804122925, 0.1642967164516449, -0.6304329037666321, 0.5488670468330383, -1.608807921409607, 0.22582559287548065, -0.5911720395088196, -0.9102423787117004, 0.6335324645042419, 0.15118610858917236, 0.787142276763916, -1.1409742832183838, -0.11912664771080017, 1.6269975900650024, 1.1758654117584229, 0.11130288243293762, -0.6255736351013184], [0.261726051568985, 1.6991246938705444, 1.9560749530792236, 2.370256185531616, 0.19357721507549286, -0.275463342666626, -0.6507411003112793, -0.9741504788398743, 0.09302064031362534, 0.35788893699645996, 0.31093189120292664, -0.7526128888130188, 1.367469310760498, 0.9246430993080139, 0.3298206627368927, -0.7577047348022461, -0.2789147198200226, 0.8344583511352539, -0.9052800536155701, -2.172903537750244, -0.16110046207904816, -0.18233349919319153, 0.05645175278186798, 0.3616868555545807, -0.10203126072883606, 0.6680330038070679, -1.7598322629928589, -0.053430844098329544, 1.1251976490020752, -1.0443339347839355, 0.8985487818717957, -0.0024420092813670635, 1.2628144025802612, 1.2452272176742554, 1.7657092809677124, -1.1543893814086914, 0.1238320991396904, -0.9979005455970764, 1.8323254585266113, 0.40588030219078064, 1.9819576740264893, 1.46449875831604, -0.749068558216095, 0.9031660556793213, -0.05701296404004097, -0.3549093008041382, 1.3561811447143555, -0.05397264286875725, 0.6340465545654297, -1.1126614809036255, 0.6440576910972595, -2.1941051483154297, 1.1001174449920654, -2.3840954303741455, 0.547313928604126, -1.338625431060791, -0.0023693593684583902, -0.40157416462898254, 0.21214355528354645, 0.33086198568344116, -0.44876623153686523, -0.19113031029701233, -0.2934570908546448, 0.8308447003364563], [-0.5619470477104187, 0.14023056626319885, -1.8962849378585815, -0.2078871876001358, -1.1526844501495361, 0.5587553381919861, -1.545168161392212, -0.11338882893323898, 1.3471845388412476, 0.7681679725646973, 0.6790848970413208, 0.4750085175037384, 0.9366756081581116, 1.7432057857513428, -0.4757435917854309, 0.963503897190094, -0.9218981266021729, -0.22844825685024261, -0.15114353597164154, 0.573602557182312, 0.9133011102676392, -0.3573925197124481, -0.0368032306432724, 1.0638337135314941, 1.0095494985580444, -0.48279693722724915, 0.7296251654624939, 0.900396466255188, 2.404995918273926, 1.0396461486816406, -0.0757250040769577, -0.6981900930404663, -1.206887125968933, 1.9443869590759277, 1.0650510787963867, -0.702560544013977, 0.9314997792243958, 0.17730222642421722, -0.7181124091148376, 0.27847203612327576, 0.09817378968000412, 0.6379702091217041, 0.49206382036209106, 1.166809320449829, 0.7767561078071594, 0.9998209476470947, -0.5967890024185181, -0.6573460102081299, 0.842624843120575, 0.17602401971817017, 0.8190132975578308, 0.16941551864147186, 0.849344789981842, 0.4321456849575043, 0.6325234174728394, 0.3440253734588623, 0.07005511224269867, 0.8935269713401794, -0.8793867230415344, 0.3990514576435089, 0.7437357902526855, -1.293823003768921, -0.560055673122406, 1.2402913570404053], [-0.841532826423645, -1.2075918912887573, -1.9329652786254883, 1.1589233875274658, 0.9964199662208557, 0.00027629631222225726, -1.4958151578903198, 0.7948094010353088, 0.8215276002883911, 2.3486034870147705, -0.765819787979126, 1.7206653356552124, -0.2569917142391205, -0.25822243094444275, 1.6205267906188965, -0.14951492846012115, 1.4816217422485352, -0.7648979425430298, 0.26309049129486084, 0.9420647025108337, -1.4316127300262451, 2.152996301651001, 1.2500959634780884, 0.08640991896390915, -0.5337941646575928, 0.676387369632721, -0.9274649620056152, 0.4976767599582672, -1.680924654006958, 0.984083890914917, -0.9379552006721497, -1.0222277641296387, -0.5689501762390137, -1.7038995027542114, 0.41160687804222107, -1.8398491144180298, 1.3463927507400513, 0.5361887812614441, 0.9314684867858887, 0.39732059836387634, 1.3942720890045166, 1.0844866037368774, 1.059541940689087, 0.4284728169441223, -2.028632402420044, -0.6932127475738525, 0.3045555651187897, -1.3995976448059082, -0.27656418085098267, -0.910285472869873, 0.7725950479507446, 0.7089772820472717, -0.11764761060476303, -0.2877315878868103, -1.280892252922058, 1.6905062198638916, 0.36390987038612366, -0.3623628616333008, -0.9679657816886902, 0.7143396735191345, 0.13698941469192505, 0.6028581261634827, -0.09949655830860138, 0.838567852973938], [0.12915341556072235, -0.10569533705711365, 0.21436701714992523, -0.4399242699146271, 0.984237015247345, 1.0810331106185913, -1.3509172201156616, 0.45800092816352844, -1.1149704456329346, 0.5699188113212585, -0.14074201881885529, -0.28228244185447693, 1.9912383556365967, -0.8049911260604858, -0.0011296529555693269, 2.875157117843628, 1.7904329299926758, 0.26152291893959045, 0.9770070910453796, 0.4071146547794342, 0.581045925617218, -0.4573829770088196, -1.1838555335998535, 1.5777270793914795, -0.268720805644989, 0.7080616354942322, -1.0240488052368164, -1.7169461250305176, -0.8369662165641785, 0.1956654191017151, 0.251899778842926, -0.7241436839103699, -1.0478423833847046, 1.856711983680725, 0.020114993676543236, -0.8734712600708008, 0.33348768949508667, 1.660443902015686, -0.6609548926353455, 0.011234244331717491, 0.5337643623352051, 0.5808494091033936, 0.6493123769760132, -0.8818138241767883, 0.16627854108810425, -0.4699212610721588, 1.072769045829773, -0.7521178126335144, -1.9816498756408691, -0.07934936136007309, 0.4399110972881317, 1.2163066864013672, -0.36170336604118347, -0.9265797138214111, 0.876675546169281, 1.638466477394104, -2.0220115184783936, 0.3911093473434448, -1.242247223854065, -0.15364326536655426, 0.6511790752410889, 1.0811740159988403, 0.12572287023067474, 1.0009336471557617], [-0.7664491534233093, 0.3203009068965912, 2.172055959701538, 2.062530517578125, -0.8708006739616394, 0.27455270290374756, 0.5776253342628479, -0.3153000473976135, -1.348706603050232, -0.5821127891540527, -1.8593987226486206, -1.1558420658111572, -0.199000284075737, -0.9089715480804443, 0.6899271607398987, -0.29660338163375854, -0.25231048464775085, 0.9388716816902161, 0.0424727126955986, -0.5573533177375793, 0.30393752455711365, 0.4316955506801605, -0.7903950214385986, 1.238571047782898, -1.3247272968292236, 0.8977420926094055, 0.1303204894065857, 1.6448249816894531, -0.2501314580440521, -0.37529444694519043, -0.9002318978309631, -0.21795108914375305, 0.7762432098388672, 1.3797117471694946, -0.1642065942287445, 2.079843282699585, 2.1743342876434326, 1.753014326095581, -1.1239614486694336, -0.5889072418212891, 1.9322432279586792, 1.3643858432769775, 0.1430332511663437, 0.7743728756904602, -0.8543418645858765, -0.5588067770004272, -1.6740692853927612, 0.30428290367126465, 0.33630552887916565, 0.11366183310747147, -1.3010809421539307, 1.1730619668960571, -0.5579694509506226, -1.8172215223312378, -0.5964486598968506, -0.45966288447380066, 0.49505364894866943, -1.668289303779602, 1.4346797466278076, 0.23792564868927002, -2.972364902496338, -2.3086369037628174, -1.2728561162948608, 0.9550144672393799], [0.7535803914070129, 1.0499603748321533, -0.9864429831504822, -0.19821450114250183, -1.0470808744430542, -0.9486013650894165, 0.5801243782043457, 0.031392116099596024, 0.5986566543579102, -0.26355215907096863, -0.4554392397403717, -0.011596102267503738, 0.6620734333992004, 0.25029319524765015, 0.4819176495075226, 1.0200661420822144, -0.6956173181533813, 1.8014410734176636, 0.4654935598373413, -1.253568172454834, 0.5295425057411194, 0.8186711072921753, -0.4957846701145172, -1.4889012575149536, -0.8161611557006836, -0.4497523605823517, -1.832811951637268, 0.4026372730731964, -0.6400847434997559, -1.3033621311187744, 1.5106663703918457, -0.15697669982910156, -1.0476229190826416, 1.17244291305542, -0.4244196116924286, 1.2427054643630981, -0.41622522473335266, 2.2024364471435547, 0.6953020691871643, 0.08577091991901398, -0.9455034136772156, -0.15408550202846527, -0.7828227281570435, 0.20759661495685577, 0.5074231028556824, 0.9345243573188782, -0.371977299451828, -0.10446591675281525, -0.5205608010292053, -1.3990439176559448, 0.10669267922639847, 1.2120227813720703, -1.776566982269287, -1.579124093055725, -0.7147623300552368, 0.09627357125282288, 0.20072099566459656, -1.3422484397888184, -0.36369994282722473, -2.0735461711883545, 0.22521090507507324, -0.09959151595830917, 0.33874592185020447, 0.37432870268821716], [-1.280784010887146, -0.3580490052700043, -1.3579506874084473, -0.9202773571014404, 0.04247438907623291, 1.2395975589752197, -0.7835597991943359, -1.0932281017303467, 1.382999300956726, -1.261619210243225, 0.7075724005699158, -0.7002582550048828, -1.1158400774002075, 0.9969757199287415, 0.1587638109922409, -1.4308269023895264, 1.6078240871429443, -1.3438388109207153, -0.1143292710185051, -0.03899533674120903, 0.3056105077266693, 1.1293991804122925, -0.2185966968536377, 0.8064191937446594, 1.633736252784729, -0.7709025740623474, 0.863519012928009, -0.6648497581481934, -0.288372665643692, 1.9100661277770996, 0.24861428141593933, 0.631676197052002, -0.2950560748577118, 3.133262872695923, 0.5915380716323853, 0.07141232490539551, 0.4752397835254669, -1.0207946300506592, -2.54414439201355, -1.5373481512069702, -0.2870260179042816, -0.07842983305454254, -0.8706098794937134, 1.3395916223526, -0.10637550055980682, 1.474583387374878, -1.1983733177185059, -1.2311151027679443, 0.494519978761673, -0.18597890436649323, -0.4676070809364319, -2.212578535079956, 1.0782021284103394, 1.4977365732192993, -0.058062594383955, 0.1773376166820526, -1.4919185638427734, 0.40430721640586853, 0.3776630163192749, 1.4454127550125122, 1.0834531784057617, 0.48458781838417053, -0.4772360026836395, 1.5966241359710693], [-0.4844915568828583, -0.4727553725242615, 0.83440101146698, 1.069385290145874, -0.8224664926528931, 1.0068225860595703, 0.5942633748054504, -0.7386958599090576, -0.9891728162765503, 1.7801545858383179, 1.6420366764068604, -0.8468291759490967, -0.25748759508132935, 0.10132208466529846, -1.099277138710022, -0.2028099000453949, 1.274432897567749, -0.5346476435661316, 1.3759373426437378, -0.09720687568187714, 1.1395758390426636, -2.4596168994903564, 1.369490146636963, 0.6519590616226196, 0.17142988741397858, 0.6423389315605164, 0.1332961469888687, -0.4672420620918274, 1.6324355602264404, -0.018656473606824875, 0.8804261088371277, -0.18723545968532562, -0.08055994659662247, 2.10442852973938, -1.9277527332305908, 0.2321757674217224, 1.4116894006729126, 0.580202579498291, -1.1901962757110596, 0.8686624765396118, 0.7156762480735779, 0.36704766750335693, 1.3714450597763062, 0.11749770492315292, -0.7871189117431641, -0.39668118953704834, 0.41581571102142334, 0.8114722967147827, -0.07013630121946335, 0.5023849606513977, -0.46641623973846436, 0.8551612496376038, -1.3527568578720093, -1.39439857006073, -0.021287526935338974, 0.4131024479866028, 0.34860044717788696, -0.382834792137146, 0.19227465987205505, -1.2728410959243774, 0.2606219947338104, -0.014183897525072098, 1.697727918624878, -0.08657608926296234], [0.6952759027481079, 0.048853300511837006, -0.8977431654930115, -0.004909907933324575, 0.3315701484680176, 0.015762679278850555, -0.6709349751472473, -0.7124461531639099, -0.08022309094667435, -0.7338950634002686, 1.4785196781158447, 2.6859936714172363, 0.012949766591191292, 0.04339492321014404, 1.6279356479644775, 0.0980117991566658, -1.2501912117004395, 0.4937625527381897, 0.7361704707145691, 0.294506311416626, -0.42425209283828735, -0.41274967789649963, -0.17816321551799774, 0.5539467930793762, -0.7925758957862854, 2.8077070713043213, -1.7290759086608887, -0.15153273940086365, -0.13533832132816315, -0.22955825924873352, 0.5665102601051331, -0.018785815685987473, 0.12761728465557098, 0.19679591059684753, 0.09211789071559906, -0.14700879156589508, -1.7326921224594116, 0.01835528388619423, 0.8520562052726746, 0.055197279900312424, 0.0952017605304718, 2.614788055419922, 0.3272371292114258, -0.22316715121269226, 1.4934148788452148, -1.3093366622924805, -0.044101133942604065, -1.553497314453125, 0.020115643739700317, -0.05686590448021889, -0.0352531336247921, 0.715284526348114, -1.4419364929199219, 1.0580874681472778, -0.06328745186328888, -1.6995474100112915, -0.48689043521881104, 1.1283272504806519, 0.46501585841178894, -0.5237467885017395, 0.2425978034734726, -0.12789702415466309, -0.6953792572021484, 0.38977429270744324], [-0.22786249220371246, 0.8791677355766296, 3.0645813941955566, 1.1419509649276733, -1.4598582983016968, -0.16161884367465973, 1.417490005493164, 0.17431896924972534, -1.7132070064544678, 0.5136883854866028, 0.2627669870853424, 1.8793092966079712, 1.261086344718933, 0.9654505252838135, -0.26567333936691284, -1.2206223011016846, -0.8847228288650513, 0.47974127531051636, -1.630071997642517, 0.37415555119514465, -1.4601048231124878, 0.0055685085244476795, -0.07168714702129364, -0.8270362615585327, 1.2404990196228027, 1.2800883054733276, -1.466163992881775, -0.7903944849967957, -0.3746585547924042, 1.0825997591018677, -0.544714629650116, 0.643337070941925, -1.8080791234970093, 0.6386785507202148, 0.4430382549762726, -1.4617503881454468, -0.2753928005695343, 0.5258386135101318, -0.1370108723640442, 1.3406870365142822, 1.0133377313613892, -0.7070270776748657, -1.690679669380188, 1.5940157175064087, 1.490414023399353, -1.4132269620895386, 0.1477213203907013, -0.28656578063964844, 0.4602867364883423, 1.6743652820587158, 2.2896032333374023, -1.1948206424713135, 0.8298050761222839, -0.5461870431900024, 0.5976123213768005, 2.079460620880127, -0.7070810198783875, -0.3104178011417389, 0.03765350207686424, 0.6593762636184692, 1.2114026546478271, 0.35032668709754944, 0.5859670042991638, 0.6242973208427429], [0.7538809180259705, 1.7526594400405884, 0.004931810311973095, -1.0221401453018188, -0.8114606738090515, 1.1761020421981812, 0.676023542881012, -0.9433738589286804, 0.4922211468219757, 0.19851507246494293, 1.6724077463150024, -1.0277678966522217, 0.13427028059959412, 0.27939099073410034, 0.44264695048332214, 0.04569822549819946, 0.2901526093482971, -0.776587188243866, 1.224644660949707, -1.7524657249450684, -0.26112329959869385, 0.020944826304912567, -0.9714491963386536, 0.6145043969154358, -1.381388783454895, -1.329482078552246, -1.0791443586349487, 0.8505414128303528, -0.7742344737052917, -0.3660486340522766, -0.28849202394485474, 0.581455409526825, -0.13153380155563354, 0.44773516058921814, -0.6644532680511475, -2.4990081787109375, 1.3432894945144653, 1.0399006605148315, -0.06952155381441116, 0.1325354129076004, -0.07558228075504303, 1.2733718156814575, 0.10038021206855774, 2.6419122219085693, -0.325884610414505, 0.09585052728652954, -0.3872222304344177, 0.11090614646673203, -2.5997724533081055, -1.5005582571029663, 0.4396447241306305, 0.7890139222145081, -0.46910208463668823, -0.49598151445388794, 0.4681037962436676, 0.48037710785865784, -0.39842328429222107, -0.5232074856758118, 0.3146103620529175, 3.5255372524261475, 0.3292800188064575, -0.23959653079509735, 0.9174132943153381, 1.0805549621582031], [-0.7090349197387695, 0.07346244156360626, 0.04059408977627754, 0.7981383204460144, -0.3724660277366638, 1.5802191495895386, -1.245728611946106, -0.7659898400306702, 1.306260108947754, 0.36651352047920227, -0.15697114169597626, -0.08301398158073425, -0.5923237800598145, -0.13104262948036194, -0.36371293663978577, 0.18802452087402344, 0.028332432731986046, -0.5283284187316895, -0.24136663973331451, -0.25773105025291443, -1.0126670598983765, -0.42730289697647095, -0.794495165348053, -2.715648889541626, 0.13728033006191254, -0.2472640424966812, 0.6721065640449524, 1.045241355895996, 1.6649072170257568, 1.048812985420227, 0.4349045157432556, 1.0972692966461182, -1.2708295583724976, -0.4093051850795746, -0.5443060994148254, -0.9777754545211792, 1.3246556520462036, -0.4163256585597992, 0.9421136975288391, -0.33651214838027954, -0.14928622543811798, -1.4270527362823486, 0.6852898001670837, 0.6695018410682678, 1.7623388767242432, 1.8453474044799805, 0.2656221091747284, -1.1794192790985107, -0.4109325408935547, 0.6990827918052673, -0.16794481873512268, 2.027662992477417, -1.416460394859314, -0.28968286514282227, 0.7172533273696899, 0.6681603193283081, -0.7691960334777832, -0.504298210144043, 0.05342541262507439, -0.14091123640537262, 2.605689287185669, 1.1470859050750732, -0.34893253445625305, 0.22453880310058594], [-0.6043334007263184, 0.5411767959594727, 1.2500516176223755, 0.47014856338500977, 1.1310700178146362, -0.9253924489021301, 0.7899609208106995, -0.7000572085380554, -0.25055328011512756, -0.10026663541793823, -0.4415915012359619, -0.365779310464859, 1.5438860654830933, 0.645423412322998, 1.3450511693954468, 0.8755754828453064, -0.1758897602558136, 0.15923240780830383, 1.3744531869888306, 0.08314526826143265, 1.0105913877487183, -0.06767051666975021, -0.6403079628944397, -0.5410274863243103, -1.1604512929916382, -0.27574148774147034, 1.7176369428634644, -1.9571125507354736, -0.08401409536600113, -1.5306278467178345, -0.49670135974884033, -1.0605496168136597, 0.2678627669811249, 0.6694924831390381, -2.186610698699951, 1.2782208919525146, -0.3361741006374359, 0.5787922739982605, -0.6389015316963196, 0.35123440623283386, 0.45873135328292847, -1.992283821105957, 0.5886799693107605, 1.5122921466827393, -0.3722424805164337, 2.0552871227264404, 1.005702018737793, -0.7982088327407837, 0.5035709738731384, 0.19636893272399902, 1.4622663259506226, -0.8141641616821289, -0.8715142607688904, -0.5400230884552002, 0.886984646320343, -1.2159978151321411, 0.3058124780654907, -0.5583168268203735, 0.5884860157966614, -0.6468396782875061, -1.060360312461853, 0.5375110507011414, -0.0775248184800148, 0.1198105588555336], [-0.6532232165336609, -0.2026466578245163, 1.0965733528137207, 0.1719464510679245, -0.3550219237804413, -0.4951465427875519, 0.6786336302757263, 0.04938031733036041, 1.2453078031539917, -1.4484583139419556, -0.10897673666477203, -0.5861581563949585, -0.30957382917404175, -0.34039753675460815, -0.5375071167945862, -0.6006677746772766, -0.21099698543548584, -0.21828687191009521, -1.1104687452316284, 0.5463493466377258, -1.1911396980285645, 0.4704490602016449, 0.1865660697221756, -1.6924446821212769, 0.8689040541648865, 0.8197064399719238, 0.7847129702568054, 1.555562138557434, 1.1650230884552002, 0.8276072144508362, 0.13671036064624786, 0.2735597491264343, -0.6086248755455017, -0.41025716066360474, -0.32852962613105774, 0.8735636472702026, 0.81412273645401, -2.3731250762939453, -1.521669864654541, -0.05251828581094742, 0.3062185049057007, -1.0499041080474854, -0.9509292840957642, -1.8756524324417114, -0.3969401717185974, 0.6391463875770569, -0.22563913464546204, -0.3536352813243866, -0.2753748595714569, -0.30123499035835266, 0.9730519652366638, -1.1137760877609253, 1.2127763032913208, -1.2238143682479858, 1.9234873056411743, 0.7094371318817139, -1.6132128238677979, 1.6774121522903442, 0.4605150520801544, 0.4185199737548828, 0.253513365983963, -0.48615047335624695, 0.3976402282714844, -0.20606344938278198], [-0.9568807482719421, 0.7573211789131165, -0.29243096709251404, -1.1535038948059082, 0.7220775485038757, -1.55661141872406, -1.2571537494659424, 0.37584900856018066, 0.29769429564476013, -1.178536295890808, 0.9231849908828735, -0.9109991192817688, 0.9640486836433411, -1.2958844900131226, -1.06339430809021, 1.6786483526229858, -1.030795931816101, -1.1450530290603638, -0.47604015469551086, -2.0754315853118896, -1.7886922359466553, -2.008819580078125, 0.7612224817276001, -0.7881888151168823, -1.3815357685089111, 0.13660699129104614, 1.0396760702133179, 0.46916741132736206, 0.8902090191841125, -0.820826530456543, 0.3056090176105499, 0.9196814298629761, -0.3716549277305603, -0.38786259293556213, 1.0523483753204346, -0.7368176579475403, 0.7881492376327515, 1.4309749603271484, 1.0757973194122314, -0.7744482755661011, 0.1517380326986313, 1.4777950048446655, 1.125518798828125, -0.9545769691467285, 0.06383711844682693, 1.6882869005203247, 2.5313165187835693, 1.7247370481491089, -1.0217634439468384, -0.5660532116889954, 0.2181243747472763, 1.0176687240600586, 0.9945705533027649, 0.44645795226097107, -0.2006610780954361, 0.277871310710907, -1.2067687511444092, 1.0453490018844604, -2.2284789085388184, -0.049060311168432236, -0.9296528697013855, 0.4907022714614868, -0.24799352884292603, -0.534571647644043], [-0.7670659422874451, 1.0610977411270142, -0.45797085762023926, -1.1125093698501587, -1.181719183921814, 2.2238736152648926, 0.278678834438324, 0.3799065351486206, 1.3762043714523315, -0.11540065705776215, 0.5646954774856567, 0.5351186990737915, -1.2021653652191162, 0.3309917747974396, -0.06081744655966759, -0.1770501732826233, -0.10349540412425995, -1.7158441543579102, 0.11239306628704071, 0.44735589623451233, -0.3130725920200348, -2.06402325630188, 0.7173106074333191, -1.165508508682251, -0.2945161759853363, -2.7354445457458496, 0.8241956830024719, -1.2154470682144165, 1.2289633750915527, -0.9366565942764282, 1.373021125793457, 1.2871522903442383, -0.6690049171447754, 0.36447593569755554, -1.5972561836242676, 0.12034004181623459, 0.15979205071926117, -0.1847556233406067, -0.4910823106765747, -0.7054689526557922, 0.41048499941825867, -0.6733148694038391, 0.4538578987121582, -1.4538456201553345, -0.5494979619979858, 1.342814564704895, 0.4893357455730438, -0.1240641325712204, -0.8383169174194336, -0.6799153685569763, 0.2177535593509674, -0.8333720564842224, -0.21530179679393768, -0.11546576768159866, -0.11397844552993774, 0.9158160090446472, 2.0251681804656982, 0.7112153768539429, 1.6630502939224243, -1.7441365718841553, 0.39973366260528564, 0.02637808583676815, -0.46419674158096313, 1.0172922611236572], [-1.0353890657424927, -0.6008274555206299, -0.13748523592948914, 0.10900572687387466, 0.9884657263755798, 0.0324958860874176, 0.6433086395263672, 0.008025022223591805, -0.2782590985298157, 0.046665046364068985, 0.11835632473230362, -2.4296295642852783, -1.3149025440216064, 0.20051465928554535, -0.2091425210237503, 0.8455072045326233, 0.9415711760520935, -0.6542887687683105, 1.7537779808044434, -1.5391325950622559, 0.20543910562992096, 1.6199383735656738, -0.5994014143943787, 0.04266142100095749, -0.7436050176620483, 0.12224572896957397, -0.40876007080078125, -0.4923306703567505, 1.064153790473938, 0.07790259271860123, -0.11250220984220505, -0.25145423412323, 0.27745139598846436, -0.45429399609565735, -0.02350359782576561, -1.5392439365386963, -0.02078777365386486, 0.4133014380931854, -0.7749457955360413, -0.2780432403087616, 2.150186061859131, 0.7178722620010376, -0.5195828676223755, -0.8884444832801819, -0.04054609313607216, 0.9957582950592041, 0.00974893569946289, -0.8856891989707947, -1.1978466510772705, 1.0969266891479492, -1.0682668685913086, -2.7954256534576416, 0.38230204582214355, 0.5582115054130554, -0.10340899974107742, -0.8080126047134399, -0.4940176010131836, -0.557323157787323, -0.46747198700904846, -1.3001092672348022, 0.7804790139198303, -1.3941359519958496, -1.4972238540649414, 0.5084686875343323], [-1.5386263132095337, 0.13469406962394714, -0.2011505365371704, 0.7880474328994751, 1.3740146160125732, -0.5655263066291809, 1.338515281677246, 0.2173755019903183, -1.1085312366485596, 0.6840322017669678, 0.6615411639213562, -0.10192092508077621, -0.5309258699417114, 0.8309181332588196, -0.5933590531349182, -1.4221200942993164, -0.6163398623466492, -0.6524394750595093, 0.11344799399375916, 0.32245856523513794, -0.04849059879779816, 0.0728938952088356, 0.8668321371078491, -0.014161582104861736, 1.5899858474731445, 1.7872267961502075, -1.1599159240722656, -0.452112078666687, 0.24642516672611237, 0.07830067723989487, 0.060289740562438965, 0.33266618847846985, 0.4921800494194031, -1.9462504386901855, 1.104651689529419, 0.5531783103942871, -1.5117989778518677, 0.6788972020149231, 1.2820117473602295, -0.7976987361907959, 0.6723688244819641, 0.8539994955062866, -0.6246427893638611, -0.40089669823646545, 0.8935236930847168, 0.24522478878498077, -0.4949791431427002, 0.5345519185066223, -0.9650068283081055, -0.19309301674365997, -1.1787902116775513, -0.10237627476453781, 0.7058790922164917, -0.7526429891586304, -1.1393940448760986, -0.8823250532150269, 0.7394728064537048, -0.7709785103797913, -0.1631454974412918, -0.9050217270851135, 0.9622780084609985, 0.8566212058067322, 0.5750272870063782, -1.148959994316101], [-0.4614766836166382, 1.5860412120819092, -0.11039033532142639, 0.3149290382862091, 1.6152781248092651, -0.09617538750171661, -0.24840228259563446, 2.014101505279541, 1.1364258527755737, 0.5718311667442322, -0.38404497504234314, 0.2762119770050049, 0.4183300733566284, 1.5197649002075195, -0.9323816299438477, 0.3844054639339447, 0.20808647572994232, -0.8235633969306946, 0.3836114704608917, -0.7400822639465332, 1.6807827949523926, 1.060295581817627, 0.8449389934539795, -1.1892973184585571, 2.037242889404297, -0.18622630834579468, -1.0742117166519165, -0.7022415995597839, -1.0533093214035034, -0.842323362827301, 0.31545183062553406, -0.21470823884010315, -2.2866029739379883, 0.5276749134063721, -1.7955172061920166, -1.1193352937698364, 0.36966460943222046, -0.13578613102436066, 0.9132653474807739, -1.2427462339401245, -1.1728650331497192, -0.3896326422691345, -0.5551044344902039, -0.19857175648212433, -0.2749069035053253, 0.23323296010494232, -0.4805665910243988, -0.3543638586997986, -1.6778901815414429, -0.2187904715538025, -1.5832974910736084, -0.6195637583732605, -0.43259257078170776, 0.9215418100357056, -0.7491788268089294, 0.21803100407123566, 1.4772228002548218, 1.706422209739685, -1.27756667137146, 0.18141388893127441, 1.2185927629470825, -0.8853927850723267, 0.7161557078361511, 0.43530282378196716], [0.8448957800865173, 0.13004085421562195, 0.5867503881454468, 0.1407991349697113, 0.3840000331401825, 0.04661685228347778, -1.0322576761245728, -0.6764430403709412, -1.0488252639770508, -1.4893677234649658, 0.16411557793617249, 0.8102487325668335, -0.9831739068031311, 0.21053604781627655, 0.7726176381111145, 1.1387875080108643, 0.6987768411636353, -0.4925265610218048, 0.2840847373008728, 0.7558435201644897, 0.1016615629196167, 2.237974166870117, -1.077241063117981, 0.9695096611976624, -2.149580478668213, 1.44791579246521, -1.1325430870056152, -1.4131220579147339, 0.9663469195365906, 0.11595701426267624, 0.7493106722831726, -0.0763440951704979, 0.014649639837443829, -0.5599738359451294, 0.6234338879585266, -0.6007729172706604, 1.056990623474121, -0.6807818412780762, -1.8043525218963623, -0.8684371113777161, 1.9037528038024902, 0.8988391757011414, -0.570602297782898, -0.4318731725215912, -0.7841740846633911, -0.18906007707118988, -0.8669942021369934, 0.20493486523628235, 0.5381137728691101, 0.9494151473045349, 0.06937350332736969, -0.20829518139362335, -0.5231139063835144, 0.2408260554075241, -0.7860498428344727, 1.5454769134521484, -0.30831217765808105, -1.884962558746338, -1.2762925624847412, -1.4849156141281128, -2.018584728240967, -0.09710337966680527, 0.1973557323217392, -0.836376428604126], [-0.33381861448287964, 0.5226311087608337, 1.0903205871582031, 0.08140186220407486, -0.6341044306755066, -1.1114782094955444, -0.5275319218635559, -1.7246345281600952, 0.5676791667938232, 0.3370205760002136, -1.894230842590332, 0.3192293643951416, -0.0810229703783989, -0.7041633129119873, 0.7059117555618286, 1.312565803527832, 1.0481219291687012, 0.6538729667663574, -0.15403081476688385, 0.298863023519516, -1.1375536918640137, -1.8728992938995361, -0.5890066623687744, 0.34385940432548523, 0.10112474858760834, 1.035614013671875, -0.8706676363945007, -1.1307446956634521, 0.13674917817115784, -0.5600196719169617, 1.0433847904205322, -0.3258529603481293, -0.1732831597328186, -0.42514973878860474, 0.7711678743362427, 1.0566121339797974, -0.3086274564266205, -1.1380387544631958, 0.009399360045790672, 0.19781345129013062, 0.22566963732242584, -0.8302252888679504, 0.3051527738571167, 1.8602173328399658, 0.4971535801887512, 0.456125408411026, -0.2437397688627243, 0.29166120290756226, 0.2871643602848053, 0.9094480276107788, 0.17944666743278503, -0.1479000598192215, -0.9986934661865234, 0.7948247790336609, 0.05306873470544815, 1.0100537538528442, -0.8830829858779907, 0.3832235038280487, -2.701882839202881, 0.47650623321533203, 1.0196619033813477, -1.0659188032150269, 1.5834120512008667, -0.22364601492881775], [0.07460958510637283, -0.03531904146075249, 2.030651330947876, 0.24097837507724762, 2.0708069801330566, -0.5925187468528748, 0.7365552186965942, 2.2119174003601074, -1.4515889883041382, 1.1727242469787598, 2.6632578372955322, 3.6733810901641846, -1.2463436126708984, -0.6629307866096497, 0.4705051779747009, -1.3499451875686646, -2.0802531242370605, 0.6842131018638611, -1.0746341943740845, 1.3078107833862305, 0.79766446352005, 0.6569122076034546, -0.27318212389945984, 0.4049181044101715, -0.15967679023742676, 1.6040223836898804, 0.4388548731803894, -0.33413878083229065, 0.27669575810432434, -0.2015519142150879, 0.25783729553222656, -0.754631519317627, -0.42065390944480896, 0.6547648906707764, 1.3661904335021973, -0.22080212831497192, 0.21619845926761627, 1.9486478567123413, 3.37300443649292, -0.8208028674125671, -0.14676518738269806, -0.007506680209189653, 1.6818225383758545, -0.43042492866516113, 0.7062041759490967, -2.019681692123413, -0.4945591390132904, 0.7619914412498474, 1.3842283487319946, -1.3974093198776245, -0.20931266248226166, 0.007888827472925186, 0.40928348898887634, -0.9118144512176514, 0.8266425132751465, 0.41516485810279846, -0.6028071641921997, -0.6018308997154236, -0.8962315917015076, -1.2956150770187378, 0.2006557583808899, -0.3343031406402588, -1.6525393724441528, -0.5474043488502502], [0.28426992893218994, 0.15776413679122925, -1.7220431566238403, 0.5837333798408508, -0.3038195073604584, 1.1864147186279297, 0.5040519833564758, -0.718110203742981, 0.49319061636924744, 0.20624610781669617, -0.41280388832092285, 0.3645327389240265, 0.567379355430603, -0.11359263211488724, 0.510177493095398, -0.5308255553245544, 0.5773915648460388, -1.1991466283798218, 0.8280668258666992, -1.043266773223877, -1.5055609941482544, 1.013076901435852, 1.370155692100525, -0.026490312069654465, -1.109798550605774, -1.2421386241912842, -0.4006904065608978, -0.13438743352890015, -0.008563010022044182, -0.5135470628738403, 1.6631168127059937, -1.2935870885849, 0.039752162992954254, 0.43057841062545776, -1.2384146451950073, 0.37062588334083557, -0.08858795464038849, 0.19321686029434204, 0.42342910170555115, -0.24493129551410675, 0.3333015441894531, 0.16140438616275787, -0.4935026168823242, -0.7026219964027405, 2.160907745361328, 0.37067484855651855, -0.17550799250602722, -0.4560529887676239, -1.3562870025634766, -0.3037734031677246, -0.057634007185697556, -0.7599833011627197, -1.539575219154358, -0.838782012462616, 1.211856722831726, 0.15828929841518402, 0.4461561143398285, -0.6458651423454285, 2.223036766052246, 0.5729212164878845, -0.7042701840400696, 0.43743667006492615, 1.340801477432251, 0.11725613474845886], [0.30109119415283203, 0.6047908663749695, -0.2235788106918335, -0.892988920211792, -1.2539138793945312, -2.395130157470703, -0.5437686443328857, -0.20784252882003784, 0.11232984811067581, -1.6029348373413086, 0.4150335490703583, 0.2179354429244995, 0.7042343616485596, 2.041548252105713, 1.3731249570846558, 0.7890341877937317, 1.92172372341156, 0.31682366132736206, -0.7131704688072205, 0.9377520084381104, -0.0257352814078331, 0.7704346776008606, 0.1761341542005539, -0.6644783616065979, -1.6243252754211426, 0.40577462315559387, 0.6728312969207764, 0.07538770139217377, -1.271158218383789, 0.7264058589935303, 0.1930125504732132, 2.108806610107422, -0.026091061532497406, 0.23590628802776337, 0.2304871380329132, -0.5745464563369751, 0.8701289892196655, -0.10626135021448135, -0.981693685054779, -0.570126473903656, 1.9105323553085327, -1.0900746583938599, 1.239472508430481, -0.9812025427818298, -2.1801609992980957, -0.27085813879966736, -0.07260655611753464, -0.6043422818183899, -0.06646779924631119, 0.37408527731895447, 0.4678284227848053, 0.5319228768348694, 0.502946674823761, -1.0055804252624512, -0.5139497518539429, -0.8862009048461914, -1.1741150617599487, 1.3785221576690674, -2.1943445205688477, -0.012054037302732468, -0.11580352485179901, 0.8203925490379333, 1.0837478637695312, 0.2559017837047577], [1.065758466720581, -1.5284619331359863, -0.8704031109809875, -1.5234423875808716, -0.9985676407814026, 0.4162799119949341, -2.1812620162963867, 1.3924164772033691, 1.6244813203811646, -0.19233746826648712, -1.3263602256774902, -0.47946175932884216, 2.818465232849121, -0.3959248960018158, 0.4953276813030243, -0.05404416099190712, -0.9378116726875305, 1.4734065532684326, 1.7069339752197266, -0.8860668540000916, -0.24245978891849518, 1.6021292209625244, 0.20577438175678253, -0.6547070145606995, 1.4541043043136597, 0.12773238122463226, 0.19301725924015045, 0.5398938059806824, -1.705053448677063, -1.8856806755065918, -0.8907174468040466, 0.6120268702507019, 0.9645541310310364, -1.6228479146957397, -0.44516560435295105, -0.01551557332277298, -0.17078663408756256, -1.893181562423706, 0.5281271934509277, 1.8155008554458618, 1.5653066635131836, -0.25240278244018555, 0.29673025012016296, 1.2384307384490967, 0.9236882328987122, -0.28246814012527466, -1.4946810007095337, -1.7838568687438965, 0.4475346803665161, 1.7264947891235352, 0.21215923130512238, 1.9434492588043213, 0.8993210196495056, 0.19720666110515594, 0.6239675283432007, 0.9807060956954956, -1.6585191488265991, 0.7104112505912781, -0.5338152050971985, -0.27671048045158386, -0.44705575704574585, -0.7619194388389587, -1.504123330116272, 1.0013275146484375], [-0.664897620677948, 0.039091773331165314, -0.7996106147766113, -0.14104121923446655, -0.8894720673561096, -0.9177383780479431, 0.3667528033256531, -0.9154656529426575, -0.39273297786712646, 1.1906559467315674, -1.2377793788909912, 0.8041322827339172, -0.6058533191680908, -0.2130867838859558, -1.0046167373657227, -1.1009571552276611, -0.6908328533172607, -1.124143123626709, -0.5626301765441895, 0.40158456563949585, 1.2712773084640503, -0.8228783011436462, 0.27120354771614075, -1.241695761680603, 0.21427525579929352, -0.5360572338104248, 0.8528515696525574, -0.34201470017433167, -1.4058963060379028, -2.3546223640441895, -0.1526421308517456, -1.2886570692062378, -1.0914852619171143, 0.5400694608688354, -0.9992502331733704, 0.9020371437072754, 2.195801019668579, 0.8501632213592529, -1.3719347715377808, 0.8802164196968079, -1.6388338804244995, -0.10946329683065414, -0.28009727597236633, 0.6202087998390198, 0.6184604167938232, -0.21453234553337097, -1.0199536085128784, 0.2101975977420807, 0.27214011549949646, 1.3253734111785889, 0.6778801679611206, -0.7829330563545227, -0.01090084109455347, -0.6389197111129761, -0.824152410030365, 0.8721791505813599, 0.13922490179538727, 0.31339284777641296, 0.2844763994216919, 0.1500866413116455, -1.9237951040267944, -0.33352744579315186, -0.7862940430641174, -1.762020468711853], [-0.8883741497993469, 0.6629833579063416, 0.2751309275627136, 0.1981029063463211, 1.308557152748108, -0.5413165092468262, 1.0156421661376953, -0.06775986403226852, -0.9244624376296997, 0.036044348031282425, 0.8482616543769836, -0.7315946817398071, -0.5566259026527405, -0.6891899704933167, -0.6123015284538269, 0.17972737550735474, 1.2975497245788574, -0.05656047537922859, -0.4180717170238495, -0.1483817845582962, 1.5550906658172607, -0.950700044631958, 1.1335612535476685, 0.9688747525215149, 0.6567038297653198, 1.617591381072998, 0.5368928909301758, 1.1032127141952515, 0.9613175988197327, -1.311703085899353, -0.6312358379364014, 0.49598702788352966, 0.03444050997495651, 0.9691221117973328, 0.08603773266077042, 0.29878753423690796, 1.1977949142456055, 0.5119683742523193, 0.8812006711959839, -0.3512480556964874, -0.7293046116828918, 0.8256510496139526, -0.16112551093101501, -1.8303579092025757, -2.5057661533355713, 0.550062358379364, 0.287471204996109, 0.18013855814933777, 0.4925244450569153, -1.2629096508026123, 0.39713990688323975, -0.12914656102657318, -0.0002520029083825648, 0.6510085463523865, 0.9716084599494934, -0.1555083692073822, -0.7356233596801758, -0.14451712369918823, 0.8683878779411316, -0.03539806604385376, 1.082028865814209, -1.664943814277649, 1.957627773284912, -0.2437143474817276], [-0.029295502230525017, 0.8980410099029541, 1.448242425918579, -1.9222420454025269, 0.04699143022298813, 1.401666522026062, 0.932988166809082, 0.7553619742393494, 0.7193418741226196, 0.03863468021154404, 0.513144850730896, -0.19027364253997803, 1.4980446100234985, 0.8131683468818665, 0.7002659440040588, -0.2935042679309845, 2.0883796215057373, 1.5375837087631226, 3.1522185802459717, -0.0869125947356224, 1.9402323961257935, -0.031702518463134766, -0.18169836699962616, -1.6151210069656372, 1.133818507194519, -1.252609133720398, -0.47879892587661743, -0.8645097613334656, -0.18688397109508514, 0.16601285338401794, -0.6979106664657593, 1.0052944421768188, -0.4048650562763214, 2.5383572578430176, 0.4497988224029541, -1.4508874416351318, 0.2234971523284912, -0.6500673294067383, -0.7110203504562378, 0.07837923616170883, -0.3809424042701721, 0.519269585609436, 1.8460092544555664, -0.7027573585510254, -0.5932686924934387, 0.46033748984336853, 0.3720290958881378, -0.025939956307411194, -0.1668507158756256, 0.09701994061470032, 0.3061579763889313, -0.0523698590695858, 0.11770899593830109, 0.006676672957837582, 1.068601369857788, -0.30225348472595215, 0.5356348156929016, 0.10877426713705063, -0.34992480278015137, -1.263264775276184, 1.2604904174804688, -1.8785556554794312, 1.385776400566101, 0.2156258523464203], [-0.5723493695259094, 1.211756706237793, -0.1417130082845688, 0.3585096001625061, -1.3075515031814575, -0.07188235223293304, -1.4489727020263672, -1.8500497341156006, -0.0225825197994709, 0.1481066197156906, -0.3509911000728607, 0.8298110365867615, 0.45625707507133484, -0.6444988250732422, 0.8997285962104797, -0.8850725293159485, -0.3515532910823822, -1.4859627485275269, 0.01337424200028181, -0.9671545028686523, 0.3853355348110199, -0.7615669965744019, 0.08925233781337738, -0.4868127703666687, 0.4602837562561035, 0.7715626955032349, 1.522317886352539, -2.1198809146881104, -0.5906865000724792, -0.7453805804252625, 0.5157643556594849, 1.696572184562683, 0.6031719446182251, 0.5030357241630554, -1.4758936166763306, -1.08006751537323, -0.614180326461792, -1.9965269565582275, -1.0903401374816895, 0.39284074306488037, -1.3249270915985107, -0.1248101219534874, -1.0795990228652954, 0.6670474410057068, -1.702847957611084, 1.5169942378997803, 1.4857527017593384, -0.2157944291830063, 0.6815783977508545, 1.0892707109451294, -0.1883271485567093, 0.17642702162265778, 0.6470485329627991, 1.2806885242462158, 0.1897372156381607, -0.567710280418396, 0.4204619824886322, 1.3849506378173828, -0.8696327805519104, 0.566451370716095, 0.2614864706993103, -0.3060382902622223, 0.8123707175254822, -3.1087522506713867], [-0.17392759025096893, -0.6248432993888855, 0.20972566306591034, -1.2525044679641724, -0.4806043207645416, 1.1023672819137573, -1.011932611465454, -1.259691596031189, 0.4756186306476593, -2.7274982929229736, -0.6059747934341431, 0.6036153435707092, -0.6001563668251038, -0.9666085243225098, -0.18761181831359863, 1.2918064594268799, -1.5320020914077759, -2.2141547203063965, 0.7394545674324036, 0.7780048847198486, 1.36322021484375, 1.223523736000061, 0.03531995415687561, 0.679276168346405, 1.224800944328308, -0.9326381683349609, -0.9869962930679321, 0.222562775015831, -0.06486718356609344, -0.17284360527992249, 0.15239116549491882, -0.5378459692001343, -0.3222825229167938, -1.0873644351959229, -1.068949580192566, 0.13016586005687714, -0.8250539898872375, 1.1451152563095093, -1.3216006755828857, -0.29227301478385925, -1.2567028999328613, -0.2524768114089966, 0.5435386896133423, 0.2404724806547165, 1.2802573442459106, 1.9859875440597534, -1.1816253662109375, 1.0251095294952393, -0.4923374354839325, -0.09373798221349716, -0.9322294592857361, 0.13175995647907257, -0.6074716448783875, 0.6560860872268677, 1.0960149765014648, -0.14488644897937775, 0.20258116722106934, 0.14579983055591583, -0.2635978162288666, 0.8996455669403076, 1.4031349420547485, 1.1519850492477417, 0.14521357417106628, 0.731982946395874], [-0.13055086135864258, 0.7111709713935852, 1.7067298889160156, 0.9303197264671326, 1.7554574012756348, -0.664792001247406, 1.634727954864502, -1.076302409172058, 0.5120895504951477, -1.7076914310455322, -1.243808388710022, 0.6452198624610901, 0.402982622385025, 0.2028679996728897, 1.112818717956543, 1.7152475118637085, -0.03966940566897392, 0.2266487181186676, 0.7775454521179199, -0.8284890651702881, -0.550616443157196, -0.4395754337310791, 1.2461856603622437, -1.6974073648452759, -0.6713048219680786, -0.08072151988744736, 0.38127562403678894, 0.5774329900741577, -0.34097886085510254, -2.610142230987549, 1.297408938407898, -1.244412899017334, -0.2616709768772125, -1.1848511695861816, -1.8066914081573486, 0.18113689124584198, 0.6766148805618286, 1.8039469718933105, -0.622668445110321, 0.1827361136674881, -0.9769880771636963, 0.46205249428749084, 0.7933858036994934, -0.8268741965293884, -0.18554995954036713, 0.574475109577179, 0.5762557983398438, 1.4535858631134033, 0.6401053667068481, 0.5789368748664856, 1.2857403755187988, -0.8557666540145874, 0.07355979830026627, 0.6233342885971069, -1.969556212425232, 0.11456035822629929, 0.5523282885551453, 0.5370501279830933, 1.7922954559326172, -0.33290737867355347, -0.8747730851173401, 0.7655237913131714, -0.0017289264360442758, -0.44165393710136414], [-0.13440227508544922, -0.1880396455526352, 0.3339669406414032, -0.2291284203529358, -0.06319881230592728, 2.6382341384887695, 0.8464630246162415, -0.40268415212631226, 1.2723065614700317, 0.08091963082551956, -0.303937166929245, -0.1447281539440155, -0.9101120233535767, -0.635024905204773, 1.2357426881790161, -0.7014488577842712, 0.6831898093223572, -1.1911267042160034, -0.30989667773246765, -0.7882506251335144, -1.5410879850387573, -1.4559253454208374, 1.6603658199310303, -0.3990515172481537, 0.25420886278152466, -0.21736125648021698, 0.5751932263374329, 0.0648895874619484, 0.9762712717056274, -0.1447567641735077, 2.1438474655151367, -0.4242338240146637, -1.169887900352478, -1.9736288785934448, -1.1924864053726196, 0.2883557379245758, 0.04411914199590683, -1.1039845943450928, -0.7348743081092834, -0.955423891544342, -0.2252565622329712, 0.1335163712501526, 2.684216022491455, -0.04900263994932175, -0.01257223542779684, -1.4136441946029663, 0.30919578671455383, -1.3903412818908691, -1.3501306772232056, -0.9265823364257812, 0.0015229224227368832, 1.3052914142608643, 0.007220360450446606, 1.3526599407196045, 0.19578862190246582, -0.7137054800987244, 2.2703192234039307, -0.0721195638179779, -0.9168896079063416, 0.43753641843795776, -0.4301374852657318, 0.7819908857345581, 1.5177212953567505, 1.0708564519882202], [0.26608920097351074, 0.2908695936203003, -1.2242456674575806, -0.8956748247146606, -0.6407802700996399, 0.2734740972518921, 0.5330759286880493, 0.18021182715892792, 0.0577135905623436, -0.8083186745643616, 0.09554368257522583, -0.05989246070384979, 0.22081758081912994, -0.7331154346466064, 0.7183175086975098, -0.40051257610321045, -0.4200325310230255, -1.4233020544052124, 0.5343233942985535, 0.2843957543373108, 0.656151533126831, 0.9430539011955261, 1.1045128107070923, -0.5521580576896667, -0.38931313157081604, -0.6142249703407288, -1.5200635194778442, -0.835141658782959, -0.33921629190444946, 1.061071753501892, 0.04853498190641403, -1.0526548624038696, 2.017374038696289, 1.5586059093475342, -1.3868488073349, 2.58597731590271, -0.5655367374420166, -0.6933470368385315, -0.9962362051010132, -1.2488207817077637, -1.6891318559646606, 0.0446101576089859, -0.9834445118904114, -0.13165253400802612, 1.2451682090759277, 0.6636437177658081, 1.7970877885818481, 2.0683434009552, 0.0658622607588768, 1.5519931316375732, 1.1834990978240967, -0.5411219596862793, -2.9370291233062744, -0.6550803780555725, -1.1687887907028198, -0.556644082069397, -0.1539931297302246, -0.4699614644050598, 1.1575371026992798, 0.14038889110088348, 0.1725892573595047, 0.4289611577987671, -0.8528138399124146, 1.7147434949874878], [-0.8275232911109924, 0.2549338638782501, 0.9702302813529968, -0.6186760067939758, -0.1059679314494133, -0.7566658854484558, -1.1227120161056519, 1.2247354984283447, 0.6901804804801941, 0.20525996387004852, -1.0160828828811646, -0.22617188096046448, 0.2909178137779236, 0.16621451079845428, -0.43841463327407837, 0.17464008927345276, 0.037261996418237686, 0.3160000145435333, -0.7225357294082642, -0.8595183491706848, -0.2723100483417511, 0.6790070533752441, 0.36555540561676025, -2.132591485977173, 0.15924382209777832, 0.8616165518760681, -0.1972510665655136, 0.08357840031385422, -1.5420745611190796, -0.9587949514389038, 1.919676423072815, -0.17151658236980438, -0.3907342553138733, -0.1847604364156723, -1.274878740310669, -0.7883780002593994, -1.9305530786514282, -0.36119717359542847, 1.6101499795913696, 0.11486788839101791, 0.34206312894821167, 0.06163296476006508, -0.6881858706474304, -0.19884520769119263, 0.8359177708625793, -0.08808554708957672, 0.8230406045913696, -0.5890282988548279, 0.485584557056427, -1.4557900428771973, 0.009148086421191692, -0.6363559365272522, -0.12639066576957703, 1.6826571226119995, 1.6030040979385376, -3.0684192180633545, -1.9132041931152344, 0.14122335612773895, 0.476850301027298, 1.5433335304260254, 0.8551596403121948, 0.7318238019943237, -1.0187420845031738, -0.4316870868206024], [-0.5257247090339661, 0.9648176431655884, 0.34276828169822693, 0.5669915080070496, 0.7741135358810425, 0.9833296537399292, 1.3187007904052734, -0.13315445184707642, 0.1958683282136917, 1.0248476266860962, 1.3758666515350342, 0.9274903535842896, 1.1252562999725342, -0.7873744964599609, 1.1457467079162598, -1.9143640995025635, 0.0036555486731231213, 1.0985734462738037, 0.275387704372406, 0.48117807507514954, 1.0806729793548584, 0.24076436460018158, 0.9189603328704834, 0.629913330078125, -1.4620389938354492, 0.15031225979328156, -0.9225218892097473, -1.3770194053649902, -0.7710782885551453, -0.27173930406570435, 1.1276477575302124, -0.4646889269351959, -1.0084151029586792, 0.29648271203041077, 2.1267197132110596, 0.7771273255348206, 0.7846265435218811, -1.0266520977020264, -0.4655050039291382, -0.6104994416236877, -1.642539620399475, 0.729658842086792, 0.2633330523967743, -0.15850232541561127, 0.6007155776023865, 0.3649519979953766, 0.490085631608963, -0.42672592401504517, -1.5645668506622314, 0.23806867003440857, -2.5613808631896973, 1.5306966304779053, 1.1851345300674438, 0.7480248212814331, 0.4000072181224823, 0.6276597380638123, 0.9908985495567322, 0.5224910378456116, -0.11934303492307663, -0.92963045835495, 0.4344750642776489, 0.3895653784275055, 3.184109926223755, -0.9330483078956604], [0.5781255960464478, 0.12392183393239975, -0.22285006940364838, -0.4163289964199066, 0.08336693793535233, 0.5810927748680115, -0.0842939168214798, 0.7878207564353943, 0.21894018352031708, 1.173862099647522, 0.1758217066526413, 0.6342272162437439, 0.283576637506485, 1.3552957773208618, -1.003538966178894, 1.3158730268478394, 0.26720768213272095, -0.9284063577651978, 0.42887571454048157, 1.1273449659347534, 0.8182843923568726, -1.6769355535507202, 0.6481781005859375, -0.9859465956687927, 0.5531596541404724, -0.7768740057945251, -0.09481679648160934, 0.9470341801643372, -0.3629997968673706, 0.7402461171150208, -1.0657049417495728, 0.8183654546737671, -0.4148230254650116, -1.6223200559616089, 0.3963924050331116, -1.0577318668365479, -1.6293714046478271, -0.37825337052345276, 2.169905662536621, 0.5044365525245667, 0.637147843837738, -0.278463214635849, -1.138291358947754, 0.9215623736381531, 0.6808743476867676, 2.468743324279785, -0.8686869144439697, -0.05064990743994713, -0.04445164278149605, -0.4608963131904602, 1.635025978088379, -0.3092391788959503, 0.3016744554042816, -0.5751309394836426, 1.13618004322052, -0.041508980095386505, 0.17860308289527893, 0.5951274633407593, 0.10427116602659225, -0.6209313869476318, 0.3738144636154175, 1.104522705078125, -0.23031851649284363, 0.7359499931335449], [-1.2127915620803833, 0.43651729822158813, 0.38283270597457886, -0.7881882190704346, 0.35011494159698486, 0.5342283844947815, 0.19520673155784607, 0.17076507210731506, -1.5662120580673218, -0.19542577862739563, 0.04235651716589928, -2.810415506362915, -0.3361071050167084, -1.144376277923584, 0.3454611301422119, 0.41211462020874023, -1.5575711727142334, 0.0028172144666314125, 0.1943756639957428, -0.9649327993392944, 1.630263090133667, -0.43580761551856995, 0.489189475774765, 0.24891798198223114, 0.591482400894165, -1.1525906324386597, 1.1211949586868286, 0.319579541683197, -0.8286803364753723, 1.616003155708313, 0.05206553637981415, -0.6601244807243347, 1.6493358612060547, -0.0836220309138298, -0.924492359161377, 1.4429941177368164, -0.5221356153488159, -0.4317391812801361, 0.6381648182868958, 0.10328911244869232, 1.0219502449035645, 0.6889551281929016, -0.4833278954029083, 1.1882151365280151, 1.0017626285552979, -0.20096735656261444, -0.5561714172363281, -0.16942644119262695, 0.14152835309505463, 0.8411166667938232, 1.7386670112609863, -0.473793089389801, -0.8137606382369995, 1.4046541452407837, -1.7100201845169067, 0.3791516125202179, 1.248084306716919, -0.9825015664100647, 0.13078515231609344, -0.809986412525177, 1.7374353408813477, 0.08553183078765869, -1.3677780628204346, -0.08924341201782227], [1.4029101133346558, -1.6524021625518799, 0.49293947219848633, 1.436087727546692, -0.3824431300163269, -1.3582624197006226, -1.165223240852356, 0.6673491597175598, 0.17039747536182404, -0.27846625447273254, 0.9005003571510315, -0.4148029685020447, 0.44172829389572144, 1.2206740379333496, 0.1776697188615799, -1.3373568058013916, 0.4277511537075043, 2.031031847000122, -0.2690980136394501, 1.2617532014846802, 1.0472792387008667, 1.4644311666488647, 0.32192060351371765, 0.4059812128543854, -0.10428769886493683, -0.5678125023841858, 0.7391191720962524, 0.6576442122459412, -0.9893015623092651, -0.11120101809501648, -1.3875266313552856, 0.4065456986427307, 0.9627393484115601, 0.07732949405908585, -0.6624400019645691, 0.6087116599082947, 0.48630937933921814, 0.16561853885650635, -0.9170971512794495, 0.6014463305473328, -0.09079644083976746, -1.3101786375045776, -1.2014636993408203, 0.12383081018924713, 0.07041598111391068, 0.15184105932712555, 0.026601966470479965, -0.7810794115066528, -0.8687517046928406, -0.22136162221431732, 0.2643839120864868, 0.25500163435935974, -0.21646633744239807, 1.6549385786056519, 0.6116700768470764, 1.0869330167770386, -0.4819791913032532, 0.9965531229972839, 1.0711997747421265, -0.44118207693099976, 1.0604044198989868, 0.8933369517326355, 1.2881401777267456, -0.6341227293014526], [0.9675757884979248, 0.2564620077610016, 0.3442457914352417, 0.5138056874275208, -2.162611246109009, 0.2040189653635025, -1.5673959255218506, 1.1110458374023438, -0.5115721821784973, 0.6811527013778687, -0.5093207955360413, 2.212541341781616, -0.8450412154197693, -1.3263829946517944, -0.9853841066360474, 0.28155890107154846, 0.33401957154273987, -0.2247050404548645, -1.3784173727035522, 0.6114393472671509, -0.5721948146820068, 0.6539351344108582, -0.14268521964550018, 1.1341373920440674, -0.8684764504432678, 0.8934531211853027, 0.32219696044921875, -0.5563457012176514, -0.11420885473489761, -0.172467440366745, -0.7707788944244385, -0.02729111909866333, 1.7860618829727173, 0.5515877604484558, 0.027100440114736557, -0.16249433159828186, -1.0543349981307983, 0.7446861863136292, -0.4487418830394745, -1.8186696767807007, 0.42864319682121277, 0.030515510588884354, 1.188440203666687, -0.8088988065719604, -1.054681658744812, 0.34325700998306274, 1.257728099822998, 0.31098082661628723, 0.2175731062889099, -0.4625815451145172, 0.7681809663772583, 0.30813470482826233, -0.2502877116203308, -1.2788398265838623, -1.7346726655960083, 1.0986392498016357, -0.9139947295188904, 0.9432785511016846, -0.9256862998008728, -0.09393977373838425, -0.08489639312028885, 1.5435470342636108, 0.6065758466720581, -0.01726517081260681], [-0.05560785531997681, 0.9947711229324341, -0.5105293989181519, 0.8287644982337952, -0.38091981410980225, -1.3792181015014648, -0.5795090794563293, 1.7396825551986694, -0.6335614323616028, -1.076924204826355, -1.142120122909546, 0.9235185384750366, 1.7517874240875244, -1.1011526584625244, -1.8634936809539795, 1.0065596103668213, 1.0565835237503052, -0.476265013217926, -1.5944242477416992, 0.9787930250167847, -0.6197074055671692, -0.870171844959259, -2.157017946243286, 0.22415325045585632, -1.6729085445404053, 0.6926697492599487, 0.2720178961753845, -2.0233395099639893, 0.9931514263153076, -0.6731446981430054, 0.8609307408332825, -0.6371920108795166, 1.669979453086853, 0.8174113631248474, -0.7893906831741333, 0.32960399985313416, -2.1463816165924072, -0.9159453511238098, -0.5407096147537231, -1.7155252695083618, 0.6060177087783813, -0.20200659334659576, 0.48622599244117737, -0.7213286757469177, 0.519039511680603, -0.5340169668197632, -0.22618694603443146, -0.7009464502334595, 0.4014508128166199, -0.30716267228126526, 1.330140471458435, -0.8937002420425415, -0.5867109894752502, 0.6480817794799805, 1.9625757932662964, 0.806639552116394, 0.30533674359321594, 1.8513977527618408, -0.26181307435035706, 1.247914433479309, 0.58005690574646, 0.019401486963033676, -1.7421000003814697, -0.0913790762424469], [0.36253830790519714, 0.18766187131404877, 0.4418872594833374, 0.05385778844356537, -1.256920576095581, 0.8821085095405579, -0.002245439914986491, -2.3817391395568848, 1.6535478830337524, 0.3660697340965271, 1.3291853666305542, 0.5935088396072388, 0.10153820365667343, 0.7399167418479919, -0.9364455938339233, -0.5061154365539551, 2.1516072750091553, 1.524303913116455, 0.19035936892032623, -1.1339963674545288, -1.4545832872390747, -0.15978245437145233, 0.055192261934280396, 1.1811163425445557, 0.5753190517425537, -0.35475414991378784, 0.520405113697052, -0.555755078792572, -0.019756732508540154, -1.6852775812149048, 0.018795179203152657, -0.02160046435892582, 0.3217969834804535, 0.638792097568512, -1.7043170928955078, 0.3503609001636505, 0.7801159024238586, -0.39290350675582886, -2.598472833633423, 0.8188138604164124, -1.3567670583724976, 0.5802518725395203, 0.02742442861199379, 1.141103982925415, -1.382948637008667, 0.7603022456169128, 0.14042237401008606, 1.6819055080413818, 0.6369701027870178, -0.3449317514896393, -0.5279456973075867, 0.03501115366816521, 0.12585866451263428, 0.25266537070274353, 0.7218632698059082, -0.9001590609550476, 0.5714482665061951, 0.42504557967185974, -0.879906952381134, -1.5867834091186523, 0.19023485481739044, -1.396176815032959, 1.5058501958847046, 0.17769001424312592], [0.9869536757469177, 0.9258952736854553, 0.693740725517273, 0.32651492953300476, 0.8195473551750183, 1.873020052909851, 0.7577750086784363, -1.0832452774047852, -0.1936386376619339, -1.3429135084152222, 0.21197502315044403, -0.7881868481636047, -0.8221966028213501, -0.9071860313415527, -0.8247765898704529, -1.0719250440597534, 1.321982741355896, 0.4955425560474396, 0.17857962846755981, -0.08477859944105148, 0.8569866418838501, -0.37613824009895325, 0.68187016248703, -0.31234124302864075, -0.7682531476020813, -1.4548271894454956, -0.8236719369888306, -0.3345884680747986, 0.7863596677780151, 0.7937242984771729, -0.08091692626476288, -0.5411351323127747, 1.2233483791351318, 2.526182174682617, 0.41905564069747925, -1.4632304906845093, 0.25571638345718384, -0.557219922542572, 2.9886224269866943, 0.8551329970359802, 0.12752893567085266, 0.49216410517692566, 1.1950887441635132, 0.43709489703178406, -0.8984799385070801, -0.9671890139579773, 1.7368435859680176, -1.512708306312561, 0.9964891672134399, -1.3485084772109985, -1.6995234489440918, -1.6860750913619995, 0.7520207166671753, 0.1020887941122055, 1.0988373756408691, -0.27381089329719543, 0.07288151234388351, 0.6168556213378906, -0.620425820350647, -0.4569738209247589, 0.5060631036758423, -1.7367680072784424, 1.320488452911377, 0.0842299833893776], [1.1165916919708252, 0.3306785523891449, 0.7947444319725037, 0.2942553460597992, 0.594113290309906, 0.6370572447776794, 0.9031373858451843, -0.5459922552108765, 1.0474464893341064, 0.0965171679854393, 0.11374129354953766, -1.151976227760315, 0.05930279567837715, 0.49426284432411194, 1.3770081996917725, -0.052444521337747574, -0.054825976490974426, 0.40455731749534607, -0.7794378399848938, 1.4246028661727905, 0.8419685959815979, -1.2773594856262207, 0.1684771031141281, 1.3166648149490356, -0.15384003520011902, -0.526012659072876, 0.9793975353240967, 0.24361690878868103, -0.1994585245847702, -1.1231107711791992, -1.3521196842193604, -0.7531417608261108, -2.683835744857788, 0.8776595592498779, -0.8282948136329651, -0.7190681099891663, -1.069029688835144, -0.9463370442390442, -0.1011708453297615, 1.1679251194000244, 1.6312347650527954, 1.2909408807754517, -0.374435156583786, 0.16677391529083252, -0.6591492295265198, -1.166581392288208, -1.7390739917755127, 1.4325952529907227, -2.2145323753356934, -0.28843265771865845, 1.3971365690231323, -0.5527530908584595, 0.5740513205528259, 0.7801864743232727, -0.954941987991333, -1.7529436349868774, 0.4731020927429199, -0.10913825035095215, 0.7303155064582825, 0.5140907168388367, -0.45742955803871155, -1.0821738243103027, 2.5141241550445557, 0.073122538626194], [0.6159871220588684, -0.7232838273048401, -0.3625154495239258, 1.294083595275879, -0.6771219968795776, -0.4984744191169739, -0.2730891704559326, 1.5148576498031616, 0.1585390418767929, 0.403728723526001, -0.03653864935040474, -0.16213752329349518, 0.8468326330184937, -0.7105810642242432, -0.06304360181093216, 2.3959057331085205, 1.552544116973877, -0.32573533058166504, -0.397174209356308, 0.7468354105949402, -0.28707247972488403, -1.5789165496826172, -0.7843543887138367, -0.1363019049167633, -0.7670816779136658, -0.18658651411533356, 0.4792976379394531, 0.03066389635205269, 0.047891415655612946, -0.7736710906028748, 0.2869485914707184, -2.1090774536132812, -0.3054448366165161, -0.8610212802886963, 0.013795278035104275, -0.36642348766326904, -0.016969842836260796, 1.292293667793274, -1.9292699098587036, -0.6942893266677856, -1.5086917877197266, -0.301383376121521, 0.5446259379386902, -0.3641264736652374, -0.6446844935417175, 0.8688360452651978, -1.6233676671981812, -1.3516846895217896, 0.06476426869630814, 1.0684189796447754, 0.6148582100868225, -0.7072116732597351, 0.20127630233764648, 0.13139501214027405, -0.3110291063785553, 0.1242794319987297, -0.7651582956314087, -0.6394175887107849, 0.3769078850746155, 1.604632019996643, 0.7078812718391418, -0.7882989645004272, -0.7146190404891968, 1.6698224544525146], [1.947827935218811, -0.27552908658981323, 1.2031303644180298, -0.2393598109483719, -2.309596538543701, 2.1582486629486084, -1.5109853744506836, 0.614784300327301, 1.0565217733383179, 0.0777677521109581, -1.4699313640594482, 1.328803539276123, -1.6433558464050293, 1.4088928699493408, -0.25011011958122253, -0.5806671380996704, 1.45039701461792, 1.4666733741760254, -1.7883007526397705, 1.9481137990951538, -0.31393560767173767, -0.8580494523048401, 0.8047376275062561, 0.01616462878882885, -0.4549506902694702, 0.8137708902359009, 0.23477783799171448, 2.187213659286499, -1.2346198558807373, -0.07195082306861877, 0.10584088414907455, 1.6431658267974854, -0.7879069447517395, 0.21328087151050568, -1.0797220468521118, 0.9544362425804138, -0.48975980281829834, 1.336426019668579, -0.2449401170015335, -0.0043692332692444324, 0.15274783968925476, -0.2502323389053345, -0.292754590511322, -0.8599118590354919, 0.8361479640007019, -2.027991771697998, 0.06800968945026398, 0.7827702760696411, -0.0532308928668499, 0.19125637412071228, 1.3172683715820312, 1.120164155960083, -0.045882221311330795, -0.5357052087783813, -1.2241424322128296, -0.15455423295497894, 1.6279902458190918, -0.8244305849075317, 1.7593679428100586, 0.668363094329834, -1.0877896547317505, 1.230413556098938, -1.2936209440231323, -1.4056192636489868], [-0.4712212085723877, -1.840157389640808, 0.15073813498020172, 0.09971974790096283, -0.5864389538764954, 0.1516389101743698, -0.12509675323963165, 0.7150041460990906, -1.9800004959106445, -2.166483163833618, 1.6093356609344482, -0.9802867770195007, -0.5990877747535706, -1.2706652879714966, 0.985940158367157, -1.8976619243621826, -1.4548951387405396, 0.27535486221313477, -1.1616461277008057, 1.911280870437622, -0.6149395108222961, -0.6102140545845032, 1.4696358442306519, 2.3601531982421875, 0.3238500952720642, -0.6803320646286011, -0.885766327381134, 0.16547980904579163, 1.0014262199401855, -1.2133183479309082, -0.414737343788147, 0.06193389743566513, -1.446914792060852, -0.7169457674026489, -0.006762478034943342, 0.535224974155426, -0.07697198539972305, 1.1768702268600464, 1.1873599290847778, 0.8374946117401123, 0.7854722738265991, 1.0908931493759155, -1.7088613510131836, -1.3654110431671143, 0.0354025699198246, 0.37897181510925293, 0.9890986084938049, 0.4450150430202484, -1.2367836236953735, 1.270190715789795, 0.1927935779094696, 0.2949744462966919, 0.32099905610084534, 0.2127510905265808, -1.3299355506896973, -0.3068702220916748, 0.6053195595741272, -0.6960275173187256, 0.1786329448223114, 0.6362076997756958, 1.060203194618225, 0.7206782698631287, 1.436098575592041, 1.478896141052246], [0.40892931818962097, 1.2526121139526367, 1.0139665603637695, -0.09249568730592728, 0.6057199239730835, -0.7985232472419739, 0.9687128067016602, -3.7620487213134766, -0.761487603187561, 1.5509693622589111, -0.10935092717409134, 0.5629677772521973, -1.0385825634002686, -1.4963289499282837, -0.11305493861436844, -0.9855486750602722, 1.034324049949646, -0.5530868172645569, 1.7909146547317505, 0.6078879237174988, -0.19795608520507812, 1.2015351057052612, 0.7310413122177124, -0.9489731192588806, -0.8615611791610718, 1.0078541040420532, -0.5830021500587463, -1.6918193101882935, 0.6868487000465393, 0.5936174392700195, -1.4209811687469482, 1.2138525247573853, 0.9642558693885803, 0.30046483874320984, -0.41235417127609253, -0.32620376348495483, 1.5482380390167236, -0.6098922491073608, 0.4805394113063812, 1.4353934526443481, -0.021694229915738106, 0.26602834463119507, -0.8681969046592712, 0.07119523733854294, 0.3739393353462219, -0.6162907481193542, -0.5636518597602844, -0.33622702956199646, -0.2340145707130432, 0.15944059193134308, 0.2864886224269867, -0.09916931390762329, -1.468470811843872, -0.2233223170042038, -1.0253453254699707, -0.4035719037055969, 0.6886455416679382, -1.2087109088897705, 0.1940692514181137, 0.017693081870675087, 1.0592573881149292, -3.0334534645080566, 0.02206556312739849, -2.0970349311828613], [-1.5998245477676392, -1.957180142402649, -1.078350305557251, 0.23625239729881287, 0.47380539774894714, 0.2946179211139679, -0.19177484512329102, 1.0961111783981323, -1.660358190536499, 1.450837254524231, 0.9575588703155518, -0.3907659351825714, 0.8676638603210449, 2.610259532928467, 0.33033299446105957, -1.7001928091049194, -1.0858166217803955, -0.021032657474279404, -0.17457175254821777, -0.08444038778543472, 1.2100425958633423, 0.2911157011985779, -0.04365485534071922, 0.5528199672698975, 0.3635663092136383, 1.5363372564315796, -1.630143642425537, -2.5964183807373047, -0.43727079033851624, 0.26467594504356384, 2.131988763809204, -0.065655417740345, 0.6479098796844482, 0.9369295239448547, 0.5722337961196899, -1.6258231401443481, -0.5530393123626709, 1.253808617591858, 0.3020530045032501, -1.1704341173171997, -1.348152995109558, -1.1106712818145752, 2.001112222671509, -1.9058408737182617, 0.6310720443725586, -1.0176602602005005, -1.5131627321243286, -2.7088234424591064, -1.1875683069229126, 0.7556223273277283, 0.8018078804016113, -1.510717749595642, 1.8188292980194092, -0.6745330095291138, -0.18209780752658844, 0.23534584045410156, -0.058673493564128876, 0.19720719754695892, -0.3845292329788208, 1.651766300201416, 2.3337953090667725, 1.0427743196487427, -1.8964420557022095, -0.4236370325088501], [-0.4925944209098816, 0.6842476725578308, 0.14874543249607086, -1.3500769138336182, 2.0608832836151123, -0.6007227897644043, -0.3426593542098999, -0.18167704343795776, 0.25907450914382935, -0.18308518826961517, -0.15534855425357819, 1.720905065536499, 0.36365601420402527, 1.3253649473190308, -1.1054641008377075, -0.5945824384689331, 1.060813546180725, -0.06750942766666412, 0.6434361934661865, -0.12538479268550873, -1.7840126752853394, -2.1521918773651123, 0.8308191299438477, -1.2251442670822144, -0.854849636554718, -0.13613151013851166, -0.1703568994998932, 0.3768720328807831, -0.6816216707229614, -1.2359615564346313, 0.3270941972732544, 0.20021401345729828, 1.2965044975280762, -0.31216374039649963, 0.6948977708816528, -0.6952710151672363, -0.42408376932144165, -1.3567636013031006, -1.5719456672668457, 0.03112575225532055, 0.868827223777771, 1.4683033227920532, -0.14531880617141724, -0.0938306599855423, 1.1114225387573242, -0.58828204870224, -0.4788362383842468, 0.7380954027175903, -1.3285775184631348, -0.6806419491767883, 1.2074388265609741, -0.32429012656211853, 0.7768257856369019, -0.7672574520111084, -1.8567763566970825, -0.6426762342453003, -1.0335546731948853, -1.5877889394760132, 0.6794987916946411, 0.6036351919174194, -0.63053959608078, -0.28515806794166565, 0.7316417694091797, -0.2378680408000946], [-0.2181219458580017, 0.24331848323345184, -1.0586059093475342, -0.5576448440551758, 0.3266235589981079, 0.2658696174621582, -0.09248745441436768, 0.9480608701705933, -0.9119458198547363, -0.49108558893203735, -0.7980262637138367, 0.3828226923942566, -1.9478120803833008, -0.5923667550086975, -0.9349948167800903, -1.4402233362197876, -0.3151061236858368, -0.15176105499267578, -0.29211050271987915, 3.0546875, -0.261983186006546, 1.290065050125122, 0.7802911400794983, 1.0919486284255981, 0.035012874752283096, -0.8682473301887512, 0.22578056156635284, 1.3604426383972168, 0.38016462326049805, 1.52654230594635, 0.7226503491401672, -0.12718601524829865, 0.005464473739266396, 2.2076361179351807, 0.6424297094345093, 0.2966294586658478, 0.5287531614303589, -1.7792179584503174, -0.6292057633399963, -1.6506160497665405, -1.638598084449768, -1.4121628999710083, 0.6159942746162415, -1.8598535060882568, 0.9359133839607239, -0.3052273094654083, 0.3932059407234192, -0.24931932985782623, -1.8666276931762695, 0.3648490011692047, -0.7484596371650696, -0.8144979476928711, -1.2999695539474487, 0.5649470090866089, -0.833088755607605, 0.6124060153961182, -1.2245090007781982, -0.43287110328674316, -0.12405507266521454, -0.38156262040138245, 0.8976207375526428, 0.2789624333381653, 0.45304980874061584, 0.3347338140010834], [-0.7587686777114868, -0.3038597106933594, 0.11408939957618713, -0.07461968809366226, -0.03376500681042671, -1.0013797283172607, 0.110076904296875, 1.8345963954925537, -0.43659064173698425, -0.2016955465078354, -0.9767062067985535, 1.1294434070587158, 1.1498030424118042, 1.0823957920074463, -0.7087457776069641, 1.6560461521148682, -0.49258938431739807, -0.8848530054092407, -0.4035896956920624, 0.10334168374538422, -0.2402358055114746, 2.259476900100708, 1.1460506916046143, 1.010684847831726, -1.3546665906906128, -0.9911670684814453, -0.46676987409591675, -0.9589825868606567, -2.2134604454040527, -2.5068652629852295, 0.4887782037258148, 0.9892759323120117, -1.1360008716583252, 1.1837666034698486, 0.04133855551481247, -0.6674799919128418, -0.5500823259353638, 1.4034132957458496, -0.88332200050354, 0.6980617046356201, -0.32514023780822754, 0.46848857402801514, -0.3240656554698944, -1.2522304058074951, 0.019143419340252876, 1.1395658254623413, -2.1366965770721436, 1.5945602655410767, -0.1745905876159668, -2.0979580879211426, 0.08135440200567245, 0.10295046120882034, -0.5161104202270508, 0.7404256463050842, -0.3458084166049957, 2.0812535285949707, -0.05634232610464096, 1.6283094882965088, -0.8414627909660339, -0.8239286541938782, 0.9627493023872375, 0.1660391092300415, -0.9646738767623901, 0.546153724193573], [-1.4445050954818726, -0.35174211859703064, -1.6956151723861694, -1.7561671733856201, 0.5986217260360718, 0.3337673544883728, -0.639814019203186, 0.029237670823931694, 0.7383407950401306, -0.9490593075752258, 0.8240959644317627, 1.7599025964736938, 0.12312375009059906, -1.4066009521484375, 1.5085583925247192, 0.1815277487039566, 1.0027192831039429, 0.7263379693031311, 1.891999363899231, -1.1479215621948242, 1.2226512432098389, 1.1129324436187744, -0.47281017899513245, -0.14459100365638733, -0.7205649018287659, -1.0140790939331055, -2.066582202911377, -0.6677188277244568, 0.5385958552360535, -0.603850781917572, 0.7204422950744629, -0.9362592697143555, -0.6009137630462646, 0.8584868311882019, 0.5213557481765747, 2.5234546661376953, -0.26882562041282654, -0.41727176308631897, -1.065405249595642, 0.36541029810905457, 1.4907195568084717, 1.094690203666687, -1.923930048942566, -1.6555501222610474, 1.0561819076538086, 0.20401489734649658, 0.8573660254478455, -0.43814077973365784, -1.1355177164077759, -1.2924041748046875, -1.2270982265472412, 0.5478215217590332, -0.12082643806934357, 1.379425287246704, 0.30565956234931946, -1.529927372932434, -1.0240411758422852, -0.07678449153900146, -0.33173295855522156, -2.269899845123291, -0.16842025518417358, 0.40596064925193787, 0.522248387336731, 0.29712310433387756], [0.39667609333992004, -0.9576423168182373, 0.17512348294258118, 1.096976399421692, -0.11080110818147659, 0.3283529579639435, 0.9498898386955261, -0.43741536140441895, 0.8696277141571045, -1.5969254970550537, 0.7013879418373108, -0.6268169283866882, 1.2168052196502686, -0.1853465586900711, -0.574554443359375, -0.16295981407165527, 0.030347313731908798, 0.4782523214817047, -1.3486876487731934, -0.9779999256134033, 0.49565795063972473, 0.732056200504303, -1.0876215696334839, -0.9269795417785645, -0.1419214904308319, -0.9530788660049438, 0.058846332132816315, -0.21817432343959808, 1.379980206489563, -0.007544645573943853, 1.805543303489685, -1.8585903644561768, -0.9699212908744812, -0.7124765515327454, -1.667745590209961, 1.8695282936096191, 0.16979572176933289, -1.0144741535186768, -1.0284923315048218, -1.0368348360061646, -1.558919906616211, -1.1003248691558838, -0.3804318606853485, -0.4385209083557129, -0.1885678917169571, 0.21603631973266602, 1.731418251991272, -0.15237008035182953, 0.11866740137338638, 0.06374915689229965, 2.7143919467926025, -1.6989809274673462, -0.6675450801849365, -0.3041170537471771, -1.9219157695770264, 0.009177381172776222, -0.09104438126087189, 1.2152247428894043, -0.18458621203899384, 1.5383614301681519, -0.2941480278968811, -0.46611320972442627, -1.474351167678833, 1.0817290544509888], [0.0839487686753273, 0.1874043047428131, 0.25549617409706116, 0.35464566946029663, 1.5657238960266113, -0.48518913984298706, 0.5642291903495789, 0.3180699050426483, -0.9517186880111694, 0.8723660707473755, 1.1858444213867188, 0.13524650037288666, -1.5882785320281982, -1.0657368898391724, -1.4008268117904663, 2.95512056350708, 0.6277593374252319, 0.6303935647010803, -0.223412424325943, -0.7205823063850403, 0.6905519962310791, 1.0224127769470215, 0.1475190818309784, -0.21762137115001678, -0.30801448225975037, 0.025048626586794853, -0.40531715750694275, 0.8946260213851929, 0.17387337982654572, -1.305299997329712, -1.3169938325881958, 0.8638971447944641, 1.0638980865478516, 0.5045480728149414, 1.6021041870117188, 0.47265592217445374, 0.20982512831687927, -0.09635843336582184, 0.4623372554779053, 0.6150256395339966, -0.27877265214920044, 0.7567989826202393, 0.5140807032585144, 0.20681270956993103, -0.29552945494651794, 0.9261425137519836, 0.5292072296142578, 1.8543280363082886, -0.450400173664093, 0.32053840160369873, 0.8326836228370667, -1.303398609161377, 0.08166838437318802, 0.23533466458320618, -1.1408461332321167, 0.8932650685310364, 0.0007107454002834857, -0.6818839311599731, -0.5087043642997742, -0.05615760758519173, 0.8884999752044678, 0.9014739990234375, 0.6939226388931274, 0.025234786793589592], [0.2872456908226013, -0.3850972652435303, -0.7394644021987915, 1.4980740547180176, -0.018382955342531204, 0.8557609915733337, 0.1169220432639122, -0.49263685941696167, 2.169004201889038, 1.2521932125091553, 0.5484636425971985, 0.17284037172794342, -0.4823044240474701, -0.3608132004737854, 1.4329770803451538, 1.42794930934906, -0.630341112613678, 0.2064114362001419, -1.0662097930908203, -0.5863693356513977, 1.8791927099227905, 2.0143685340881348, 0.9908562898635864, 1.4639201164245605, 0.1361861675977707, -0.368605375289917, 0.5378670692443848, -2.8552474975585938, 1.5980720520019531, -1.3837381601333618, 1.8951665163040161, -1.0874991416931152, 0.003369402838870883, -1.9004390239715576, 1.4865093231201172, 0.37612172961235046, -0.3973909914493561, 0.2960135042667389, 0.03714701905846596, 0.45636239647865295, -2.407118558883667, 1.5560601949691772, 0.32148852944374084, -0.2360057234764099, -1.7436658143997192, -0.7590453624725342, -2.2278027534484863, -0.41104114055633545, 0.5208393335342407, 0.704103410243988, -0.477017343044281, 0.310647189617157, -1.4270809888839722, 1.6604055166244507, -1.153315544128418, -0.6440831422805786, -0.049073755741119385, -1.178725004196167, 0.8090968728065491, 2.6752233505249023, 0.8923813700675964, -0.738078236579895, 0.20629574358463287, -0.31530407071113586], [-0.7685322761535645, 0.4189628064632416, 0.4728143811225891, -0.042058926075696945, -0.8939419984817505, 1.1273868083953857, 1.3910284042358398, 0.7955954670906067, 0.5591276288032532, 1.7714869976043701, -1.0159153938293457, -1.7466588020324707, 0.23213648796081543, -0.8529662489891052, -0.22170650959014893, -0.2856172025203705, -1.1222774982452393, -1.7488001585006714, 0.6045972108840942, 0.16683745384216309, 0.474968284368515, -0.5423191785812378, -1.075223445892334, -1.2159578800201416, 0.4942401349544525, 1.2484029531478882, -0.8554272055625916, 0.09228197485208511, 0.0495401993393898, 0.3313782513141632, 0.2305920273065567, -0.1872001439332962, -0.4718157947063446, -2.1635217666625977, -0.11218443512916565, 0.9605724215507507, -1.390820860862732, -0.1817404180765152, -0.7471885085105896, 0.9803689122200012, 0.6759051084518433, -1.2145129442214966, 0.18010449409484863, 0.11151903867721558, 0.4746430814266205, 0.08745381236076355, -1.5487878322601318, 1.9535348415374756, 0.9396551847457886, -0.10090978443622589, 0.9652284979820251, 0.5051686763763428, -0.7352043986320496, 0.5901428461074829, -0.074558787047863, -2.554462432861328, -0.7299404144287109, -0.4786287546157837, 0.6292682886123657, 0.18836435675621033, 1.6229339838027954, 0.20029042661190033, 1.1143865585327148, -0.02504844404757023], [1.88798189163208, 0.7405152320861816, -0.7443186044692993, -1.2267982959747314, -0.893510103225708, -0.6222847700119019, 0.1460077166557312, -0.37046194076538086, 0.048630788922309875, 0.07786400616168976, 0.5333929061889648, -0.3713051378726959, -0.3607800602912903, 0.44628870487213135, -1.321041464805603, 0.12963351607322693, 1.080617070198059, -1.3215175867080688, 2.569049119949341, 0.48510944843292236, 0.33838024735450745, -0.4639661908149719, -1.0367891788482666, -0.7100721597671509, 0.08433747291564941, -0.5277175307273865, 0.9622592926025391, 0.2006673663854599, -0.16453449428081512, 0.23875725269317627, 0.9021457433700562, 1.8563661575317383, 0.7781447172164917, 0.18807336688041687, 0.34864580631256104, -1.3659076690673828, -0.48928627371788025, -0.5648703575134277, 1.1770713329315186, 2.6026675701141357, 0.5190407037734985, -1.8288861513137817, -0.022818060591816902, 0.3392695188522339, 1.5529859066009521, -0.8665871620178223, -0.6721107363700867, -1.7231721878051758, -1.848682165145874, 0.8418987989425659, 0.9604986310005188, -1.3683840036392212, -0.34608688950538635, -0.7754723429679871, 0.9168148636817932, -0.7500126361846924, 1.289276123046875, -0.8929477334022522, 0.49478065967559814, -0.8510754704475403, -0.5673924088478088, 0.8015017509460449, 1.0968493223190308, -0.18678495287895203], [-1.0349527597427368, -2.9038562774658203, -0.9131157398223877, -0.06867336481809616, -1.0978337526321411, 1.0784828662872314, 2.6624093055725098, 0.8923593163490295, 0.5275177359580994, 1.1184041500091553, 0.9812986850738525, -0.20080561935901642, 1.7677396535873413, -0.29530248045921326, -0.6923308372497559, 0.17046110332012177, -0.5036383271217346, -1.7062541246414185, 0.5135678052902222, -0.34754571318626404, 1.5362162590026855, 0.9837929010391235, 1.0142192840576172, -0.46853405237197876, -0.6108986735343933, 0.3919508457183838, 1.0797525644302368, 0.4126769006252289, -0.8551352024078369, 0.3198099434375763, -1.2147541046142578, 0.32977309823036194, 1.969906210899353, -0.3641127645969391, 1.9984310865402222, 0.8775292634963989, -0.11000845581293106, -2.055793523788452, 0.9893555641174316, 1.626399040222168, 0.7348012328147888, 0.5174026489257812, 0.9664311408996582, 0.3502460718154907, 0.732722282409668, 0.8622181415557861, 0.7762159109115601, 0.6854131817817688, -0.6010012626647949, -2.6849751472473145, 0.593258261680603, 1.6258573532104492, 1.8410943746566772, 0.9548982977867126, 1.1577414274215698, -0.030923211947083473, 1.0231337547302246, -0.24065425992012024, 1.2072696685791016, 0.10598499327898026, -0.5715395212173462, -0.626326322555542, -0.7386807799339294, 1.715451717376709], [0.6042587161064148, -0.010431172326207161, 0.7420270442962646, -0.3962312638759613, 0.6977759003639221, 0.8153069019317627, 0.987322211265564, 0.9292560815811157, -1.2751529216766357, -1.4244798421859741, 0.1719825118780136, 0.7159608006477356, 0.41728129982948303, 0.9852802753448486, 1.3705366849899292, 1.4767969846725464, -0.09656096994876862, -0.6790695190429688, 0.4644322395324707, -0.11344210058450699, 1.36383855342865, 0.037449248135089874, 1.099819540977478, -1.4183074235916138, 0.4652753174304962, -0.9042404294013977, -0.4339280426502228, 0.5235033631324768, 0.4865938127040863, -1.5003113746643066, -0.3735465705394745, 1.073500633239746, -0.03575578331947327, -0.25039178133010864, -0.024767139926552773, -0.24801991879940033, 0.36258959770202637, -0.7914522290229797, 0.7943857312202454, 1.2754629850387573, -1.6122081279754639, -0.21766279637813568, 0.9346680641174316, -0.5676090717315674, -0.3081057369709015, 0.28016898036003113, 2.3021998405456543, 0.20755943655967712, 1.2024762630462646, 0.18704161047935486, 0.13011205196380615, -0.04587482288479805, -0.6770921945571899, 0.3821857273578644, 0.2845193147659302, -0.9708619713783264, 0.4348563849925995, 2.0716614723205566, 0.20260077714920044, -0.38940346240997314, 1.040356159210205, 0.3267739713191986, 0.447924941778183, 0.1975286453962326], [-1.5441800355911255, -1.2798445224761963, -0.738953709602356, -0.2630687654018402, 0.06385244429111481, -1.5686253309249878, -1.2634503841400146, -0.802240252494812, -0.8209148645401001, -1.5201197862625122, 1.4023381471633911, 0.49355167150497437, -0.332988440990448, 0.464145690202713, 0.18414247035980225, 0.07705190032720566, -0.8068052530288696, -0.9568371176719666, -0.43426990509033203, -0.16567377746105194, -0.533730149269104, -0.9036783576011658, -0.7986008524894714, 1.3105800151824951, -0.6113726496696472, -2.3235161304473877, -0.4419247508049011, -0.23177431523799896, -0.01972685568034649, -2.16749906539917, -0.9394425749778748, 0.3779853582382202, -0.9935755729675293, -1.439871072769165, 0.9997748136520386, 1.3563956022262573, -1.2049212455749512, 0.4623262286186218, -1.6328420639038086, 0.6926309466362, 1.205572247505188, 0.21568582952022552, 0.21580341458320618, -2.09450626373291, -0.5609867572784424, 0.7829145789146423, 0.6794024109840393, 0.30911704897880554, 1.0440199375152588, -0.6947565674781799, -0.5908897519111633, -0.5429679155349731, 1.3048292398452759, -1.2083560228347778, -0.1705957055091858, -0.30622464418411255, -1.6484981775283813, 0.9230899214744568, -2.1533517837524414, -0.7547432780265808, 0.15632112324237823, 0.6238910555839539, -0.5044831037521362, 1.332243800163269], [-0.8149453997612, 0.050651777535676956, 0.38931089639663696, -1.4099926948547363, 0.9392385482788086, 0.4721859097480774, -0.8312799334526062, 0.1224408969283104, 0.08674169331789017, -0.8990528583526611, -1.1860277652740479, -0.39530906081199646, -0.5409044623374939, -0.06773005425930023, 0.1598404347896576, 1.127910852432251, 0.26581767201423645, 0.7149099707603455, -0.4146296977996826, -0.9562271237373352, 1.4427140951156616, 0.02500840462744236, 0.18773245811462402, 0.0892714262008667, 1.2044320106506348, 2.0079660415649414, -0.5515699982643127, -1.3210196495056152, -0.8139252662658691, -0.5523731112480164, 2.3446505069732666, 0.37767910957336426, 0.9948827028274536, 1.6799767017364502, 0.09606274962425232, -0.9221051931381226, -1.729386568069458, -0.12476497143507004, 0.09248057752847672, -2.632070779800415, -0.618152379989624, -1.8318980932235718, -2.5925350189208984, -0.2465265691280365, 0.14056800305843353, -0.12653258442878723, -0.5018508434295654, -0.5904079675674438, 0.14783689379692078, 1.2253774404525757, -1.6685147285461426, 0.09893116354942322, 0.3998998999595642, -0.5713015198707581, -0.00046567170647904277, 0.6789385676383972, 0.6378235220909119, 0.38125884532928467, -0.12184411287307739, 0.8907747268676758, 0.6982028484344482, -1.9839754104614258, -1.4281084537506104, -1.1575441360473633], [0.44088029861450195, 0.0027457899414002895, -0.37860220670700073, 2.3639206886291504, -0.8514419198036194, -0.3457596004009247, 0.4976236820220947, -1.41590416431427, 0.9170891642570496, -1.0781865119934082, -0.5035696625709534, 0.12025074660778046, -0.39532867074012756, 1.2259160280227661, -0.09464346617460251, -0.9971374273300171, 0.3090786337852478, -0.38317495584487915, -0.6087930798530579, -1.1175285577774048, 0.9191381335258484, -0.17568454146385193, -0.35487818717956543, -0.41486436128616333, 0.7215505242347717, -0.06799428164958954, -0.39045432209968567, 1.2730917930603027, -0.6157607436180115, -0.956170916557312, 1.8580909967422485, 0.5096423029899597, -0.841482937335968, -2.1421422958374023, 1.3241511583328247, -0.010335434228181839, 0.7311057448387146, -1.1253050565719604, -1.4028321504592896, -0.612924337387085, -2.4778335094451904, 0.9988710284233093, 1.4583419561386108, -0.6007005572319031, 0.372051864862442, -0.4079391062259674, 1.1595336198806763, 0.29810619354248047, 0.34779787063598633, -1.2050620317459106, -1.8398982286453247, 1.252167820930481, 1.3854259252548218, -1.8165544271469116, 1.1601067781448364, 0.9493986964225769, -0.4194289445877075, -0.28001871705055237, 1.660739541053772, -0.4728355407714844, 0.37624838948249817, -1.2446786165237427, -1.6955379247665405, 0.34189242124557495], [1.1674154996871948, -0.4856180250644684, -0.08990001678466797, 0.5633920431137085, -1.09154212474823, 0.8687445521354675, 0.8234251141548157, 0.8520983457565308, 0.8972877264022827, -2.5570616722106934, -0.7725195288658142, -1.0490550994873047, 0.09958933293819427, -0.4209328889846802, 0.1600208878517151, 1.4776805639266968, -0.2895607054233551, 0.582802414894104, -0.3406640887260437, 0.42254194617271423, 3.2328057289123535, 1.0569521188735962, 1.3097054958343506, -0.485416442155838, -0.4990359842777252, 1.9220695495605469, 0.5352386236190796, -1.386250615119934, -0.9622654914855957, -0.03263361006975174, -0.563183605670929, 1.1090754270553589, -0.7064331769943237, -0.5556672215461731, -0.7090094685554504, 0.6617170572280884, -0.9156020283699036, -0.2258147895336151, -0.5839788913726807, 0.5149266719818115, -0.535409152507782, 0.034789904952049255, -0.4670906960964203, -0.7680578231811523, -0.7124056816101074, -1.0484529733657837, 0.3860435485839844, 0.44888821244239807, -1.1922098398208618, -0.9335877299308777, 0.3510289192199707, -1.0680246353149414, 2.297102212905884, 0.24901148676872253, -2.623593807220459, -0.4711211919784546, 1.350128412246704, -0.2776361107826233, -0.6710378527641296, 0.12941236793994904, -0.13365910947322845, -0.3691277503967285, 0.5357623100280762, -1.344265103340149], [-0.30010154843330383, -1.082503318786621, -1.2944469451904297, 0.6189597249031067, -0.19757433235645294, -0.7911967635154724, -1.3879417181015015, 0.4233245551586151, -0.45733797550201416, 1.7295701503753662, -0.5183074474334717, 1.3365381956100464, 0.28198373317718506, -0.5503854155540466, 0.519049346446991, -1.3514869213104248, 0.3106238543987274, 2.0712757110595703, 0.84602952003479, 0.23753251135349274, -0.13572755455970764, -0.4726971387863159, 0.08779910206794739, -0.031899143010377884, 0.6340184211730957, -0.48632627725601196, -0.4367717206478119, -0.6931440830230713, 1.237912654876709, -0.2507265508174896, -0.4478223919868469, 0.6674336194992065, 0.27378779649734497, 0.4202776253223419, 1.41648530960083, 0.6831243634223938, 0.013169596903026104, -1.1557224988937378, -0.5515178442001343, 0.41933345794677734, 0.7736924290657043, -2.005695343017578, -0.28121039271354675, 0.5042422413825989, -0.4698004126548767, 0.4229028820991516, -0.060637328773736954, -0.9685620069503784, -0.6070523262023926, 1.3951137065887451, -0.29822206497192383, -1.4083945751190186, -0.5428963303565979, 0.15179798007011414, -0.9976343512535095, 0.5916568636894226, -0.48657673597335815, -0.2701321542263031, 1.0412476062774658, -0.22908948361873627, 0.5500612258911133, 0.6856012940406799, 0.6218965649604797, -2.749720573425293], [2.2027461528778076, 0.08150860667228699, 1.2087198495864868, -0.8639803528785706, -1.0828982591629028, -0.28891876339912415, -1.125061273574829, -1.191831111907959, 0.20027408003807068, 0.5728232264518738, 0.6032254695892334, -1.1471295356750488, 0.6338215470314026, 0.946827232837677, 0.7815958261489868, -0.5398863554000854, 1.662918210029602, 0.2572686970233917, -2.2445833683013916, 1.2933608293533325, -0.4411664307117462, -0.8480983972549438, -0.5429915189743042, -0.09833786636590958, -1.4791371822357178, -1.0052357912063599, -1.2953801155090332, -0.673682689666748, 1.9154645204544067, -1.2719162702560425, -0.1281004697084427, 1.3853665590286255, 1.0680603981018066, -1.5345101356506348, -0.5392095446586609, -0.7721836566925049, -0.9943864941596985, -1.019972324371338, -1.0359792709350586, 1.331830620765686, -0.9551874399185181, 1.41886305809021, -0.5365357995033264, 0.31295880675315857, 0.9570838212966919, -2.1563572883605957, 1.4478727579116821, 0.8846800327301025, -1.1616909503936768, 0.8878558874130249, 1.6943405866622925, -1.268838882446289, -2.666308879852295, 0.46492594480514526, -0.2527855932712555, -0.22945237159729004, 0.18229073286056519, -0.7715489268302917, -1.4949620962142944, -1.2378144264221191, -0.22437703609466553, -0.5467011332511902, 0.26570239663124084, 2.4564266204833984], [1.0892276763916016, 0.8224303126335144, 0.2742909789085388, 0.46789851784706116, 0.34228771924972534, -0.9259859323501587, -0.716137170791626, -1.3002419471740723, -0.6825237274169922, 1.1010686159133911, -0.8400569558143616, -0.7092976570129395, -0.38255736231803894, -1.878792405128479, -0.28540727496147156, -0.4487420618534088, -0.021284069865942, -0.6344684362411499, -0.5442373752593994, -0.5971399545669556, 0.18027548491954803, -0.3648360073566437, 0.06581343710422516, -1.3484210968017578, 0.9489158391952515, 0.2647895812988281, 0.27538546919822693, -0.6973959803581238, -0.03386304900050163, -0.010896163992583752, 1.6833192110061646, 0.1223481297492981, 1.990161418914795, 2.1449944972991943, -0.34175539016723633, -0.8031398057937622, -0.33022820949554443, -1.322177767753601, 0.8100512623786926, -0.5109168887138367, -1.7201721668243408, -0.4553369879722595, -0.5834985375404358, 0.7114605903625488, 1.0312966108322144, 1.4870232343673706, -0.542352557182312, -0.5374382138252258, 0.48368263244628906, 0.4902057349681854, -0.4117949903011322, -0.6487747430801392, -0.24006196856498718, 0.409105509519577, 0.06369928270578384, -0.5209858417510986, 1.0208178758621216, 0.3613336682319641, -1.3559024333953857, -0.4525485336780548, -0.43341904878616333, -0.459372878074646, -2.671389102935791, 0.44852006435394287], [0.5218368768692017, 0.5084860324859619, 0.42096179723739624, -0.2876143157482147, 0.16204337775707245, -0.29544827342033386, -1.5199476480484009, -1.007768154144287, 0.6581781506538391, -1.341153860092163, -0.18576818704605103, -0.746269166469574, 0.6100069284439087, 1.9791804552078247, -0.0277867428958416, 0.7716136574745178, 1.7332789897918701, -1.4287440776824951, 0.5429946184158325, 0.6804051399230957, 0.31286734342575073, -0.20637433230876923, 1.412745475769043, -0.1538795530796051, 1.4347021579742432, -1.944265604019165, -0.01774180680513382, -0.3230176270008087, 1.8288382291793823, -0.6083860397338867, 0.780071496963501, 0.5407494902610779, -0.7662293314933777, -0.4278983473777771, -0.5444518327713013, -2.421988010406494, -0.4779839515686035, -0.7877722978591919, -0.8402875065803528, 0.5465750694274902, -0.5807691216468811, -1.1066842079162598, 0.507013201713562, 1.4071754217147827, 0.3709527552127838, 1.3807048797607422, 0.3149890899658203, 2.1198105812072754, 0.3753964602947235, -1.0976998805999756, -1.9411050081253052, -0.7798587679862976, -1.3227131366729736, 0.6869587898254395, 0.08489545434713364, 0.7273880243301392, -0.4531392753124237, 1.2128090858459473, -1.021052598953247, -1.4728341102600098, -0.014937901869416237, -0.6601905226707458, -1.69614839553833, 1.2490429878234863], [-0.6829359531402588, -0.8247010111808777, 1.476015567779541, -0.7986043095588684, 1.9926121234893799, -0.5120535492897034, 0.9574737548828125, 1.409214973449707, -0.42785438895225525, 0.7378637194633484, -0.32907938957214355, 0.2670372426509857, 0.9211045503616333, 1.653134822845459, 0.9972242116928101, 0.6821026802062988, -0.8219457268714905, -0.9207280874252319, -1.5159355401992798, 1.2702890634536743, -1.1608458757400513, -0.37370187044143677, -1.539551019668579, 0.46393120288848877, 0.46634918451309204, -0.5862675309181213, 1.4075080156326294, -0.6782464385032654, -2.9435575008392334, 0.8100863099098206, -1.012150526046753, -0.7491722702980042, -1.5431357622146606, 0.7061342597007751, 0.6353517770767212, 0.445081502199173, 0.11610595136880875, 1.2082774639129639, -0.6099950671195984, 0.2510293126106262, 0.762218177318573, 0.2538958489894867, 0.022326117381453514, -0.7105928063392639, -1.086270809173584, -1.886108636856079, -0.05917039141058922, -0.02964690886437893, 0.03534421697258949, -1.0542981624603271, 1.542946219444275, -0.2956434190273285, -0.6393142938613892, -0.5283255577087402, 0.1570887267589569, -0.08809401094913483, 0.35017114877700806, -0.3974648714065552, 0.6282516121864319, 0.3871299922466278, -1.2186697721481323, -0.8405478000640869, 0.9680575728416443, 1.3833411931991577], [0.21525923907756805, 0.00813011173158884, -1.5734217166900635, 0.25920069217681885, 1.0453864336013794, 0.6337274312973022, 0.7066226005554199, 0.06585776060819626, -0.20709963142871857, 2.0462560653686523, 0.3596987724304199, 0.5330582857131958, 0.4656684398651123, -0.022357413545250893, -0.3301302492618561, 1.7139949798583984, 0.2323143035173416, 0.872462809085846, -0.5704464912414551, 1.1387685537338257, -0.10375265777111053, -1.0237442255020142, -0.6398475170135498, 0.43925556540489197, -0.39634543657302856, 1.1549289226531982, -0.8503427505493164, 1.3856498003005981, -0.12491698563098907, -0.09944085031747818, 0.4253433644771576, 0.14243926107883453, 0.5928817391395569, -1.6235429048538208, 1.5647183656692505, -0.02630598098039627, 0.25213223695755005, -0.6060832738876343, -0.9699026942253113, 0.7747640013694763, -0.5545840859413147, -0.5770887136459351, 0.40875211358070374, -0.6933444738388062, 1.0890990495681763, 0.8191297054290771, -1.0326406955718994, -0.050595201551914215, -1.0866780281066895, -1.3151276111602783, -0.19008056819438934, 0.6320022940635681, 1.1831926107406616, -0.4120290279388428, -0.12717166543006897, -1.3709467649459839, 0.429321825504303, 1.6758785247802734, 0.339417040348053, -1.450385570526123, 1.0177315473556519, -1.044001579284668, 1.1353040933609009, -1.0066791772842407], [1.6107937097549438, -0.0900682881474495, -0.5714527368545532, 0.5676428079605103, 0.976026177406311, -0.03228989988565445, 0.39855170249938965, 1.1276421546936035, 0.7938452363014221, 1.0132721662521362, 0.40213742852211, -0.401549369096756, 0.9980387687683105, -0.3377929627895355, -0.890522837638855, -0.5239214301109314, -0.30047810077667236, -1.5251824855804443, 1.5328166484832764, -1.124945878982544, 0.08924800902605057, -2.099881887435913, -0.35931622982025146, -1.2432574033737183, 0.235518679022789, 1.709753394126892, -0.11380414664745331, 0.13546505570411682, 2.023043155670166, 0.05816545709967613, -0.27956289052963257, 0.04215875267982483, -0.7050825953483582, -0.4811836779117584, -0.5837803483009338, 1.9952436685562134, -1.1539418697357178, 0.3356257975101471, 0.1340658962726593, -1.5657087564468384, -2.3815956115722656, 0.3639332354068756, -0.5543227791786194, 0.17615045607089996, 1.1617431640625, -0.19178077578544617, -1.4849326610565186, -0.6726645231246948, -1.6029232740402222, -1.3988542556762695, -0.36922842264175415, -0.02189265750348568, 1.7771785259246826, -2.099454879760742, 0.7922360897064209, -0.04185174033045769, -0.2885505259037018, 0.9980458617210388, -0.7616941332817078, 0.4662196934223175, 1.2797467708587646, -0.4548155963420868, -1.9863696098327637, 1.2925853729248047], [-1.0204187631607056, 0.3073953092098236, -2.03383731842041, 0.5423707962036133, 0.6302054524421692, 1.213456630706787, -0.8953569531440735, 0.02602638304233551, 1.624963641166687, -1.2323546409606934, 1.3518462181091309, 0.20161034166812897, -0.11647598445415497, -0.9714158773422241, 0.8048812747001648, 1.447966456413269, -0.7787768840789795, -2.565162420272827, -0.09595008939504623, -0.5657538175582886, 0.9160211682319641, 1.6183983087539673, -1.3173965215682983, -1.7139323949813843, 0.7264648675918579, -1.1369543075561523, 0.3370814025402069, 0.9213350415229797, -0.8645851612091064, 1.1785060167312622, 0.28358975052833557, -1.1967445611953735, 1.0191184282302856, 0.30826297402381897, -1.027536392211914, -2.0753867626190186, 1.0982869863510132, -0.07090751081705093, 0.4953264892101288, -0.11075478047132492, -0.8078720569610596, -0.9528157114982605, -0.2616400718688965, 0.12899374961853027, 1.8843506574630737, 1.1971871852874756, 1.155148983001709, -2.138660192489624, 1.8220912218093872, 1.5018795728683472, -0.9196163415908813, -1.1330103874206543, 1.8105305433273315, 0.20605036616325378, 1.0996123552322388, 0.329023540019989, 0.6333886384963989, -0.8654270768165588, 0.09993185102939606, 0.06353621184825897, 1.3837858438491821, -0.41710183024406433, 1.4804952144622803, 0.6179563999176025], [-0.41032326221466064, -1.428534984588623, -0.6623958349227905, 0.8610539436340332, 0.30311957001686096, -0.2818155586719513, -0.9327948093414307, -0.4580134451389313, 1.1955721378326416, -0.7329565286636353, -1.1948579549789429, -0.9735000729560852, 1.4960541725158691, 0.7233056426048279, -0.12986324727535248, 0.07212900370359421, 0.047149769961833954, -0.6736178994178772, 1.2772003412246704, -0.6804725527763367, 0.3035537898540497, -0.3522118031978607, -0.17530585825443268, 1.059175968170166, 1.5735318660736084, 0.3960118591785431, -0.6004095673561096, 0.7403618693351746, 0.14154784381389618, -0.0498601570725441, 0.7771977186203003, -0.35705482959747314, 0.6551926136016846, -1.0542155504226685, 1.5271755456924438, 0.48894986510276794, 0.7318373322486877, 0.9452229738235474, 0.8153282403945923, -0.2355947047472, -0.9126498103141785, 1.9321690797805786, 1.7219840288162231, -2.089831829071045, 0.3262374699115753, -1.3128504753112793, 0.35935845971107483, 0.18228016793727875, -0.9261102080345154, -1.2659697532653809, -1.4187201261520386, 0.8466116189956665, -0.6006911396980286, -1.1634526252746582, -0.6333227753639221, 1.2190454006195068, -0.5592243075370789, -0.7571316957473755, -0.8001473546028137, 1.6471275091171265, 1.6369695663452148, 0.10004518926143646, 1.1748443841934204, 1.8211246728897095], [0.04590277001261711, 0.6384986639022827, 1.1372648477554321, 1.25049889087677, 0.9998153448104858, -0.4636366367340088, -0.14730069041252136, -1.4333134889602661, 0.5681618452072144, -0.3285636305809021, 1.8599640130996704, -2.1314516067504883, 1.7905385494232178, -1.6327685117721558, -0.19584661722183228, 1.033936858177185, 1.0622961521148682, -0.5493723154067993, -0.3177599310874939, -1.8069225549697876, 0.6585459113121033, 0.663649320602417, -0.8850308656692505, 0.34515181183815, 0.7798356413841248, 0.08584368228912354, -0.6775248050689697, 0.35375547409057617, 0.1302596479654312, 0.08071301877498627, 1.3079299926757812, -0.43010321259498596, 0.7509898543357849, -1.2839590311050415, 0.51793372631073, 0.24729835987091064, 0.2042284905910492, -0.5812540650367737, -0.6251245737075806, -0.3243093192577362, 0.6992052793502808, -0.8806220889091492, 1.1422019004821777, -0.3566075265407562, 0.08022543787956238, -2.3904693126678467, -2.307222604751587, 0.35617804527282715, 0.6903272271156311, 0.8822504878044128, -0.07043137401342392, -1.4319852590560913, 0.18245697021484375, -0.933164119720459, -0.12356525659561157, 0.8889656662940979, 1.793802261352539, 0.038858357816934586, 0.8922648429870605, 0.6545895338058472, -0.24017515778541565, 1.0551053285598755, 1.4443938732147217, -1.61258065700531], [1.0687073469161987, -0.04173281416296959, -0.3566926121711731, 0.5355247855186462, 0.1624699980020523, -2.9187991619110107, -0.9985232353210449, -0.42830684781074524, 0.20530809462070465, 1.3309075832366943, -0.45500269532203674, 0.7579212188720703, 1.1342421770095825, 0.7827721238136292, -1.4247016906738281, -0.6841025352478027, 0.4508236050605774, 1.3223990201950073, -0.36055970191955566, 0.6891651153564453, 1.8529466390609741, 0.39060717821121216, -0.22742730379104614, -0.4451421797275543, -0.15529009699821472, 1.281716227531433, 0.7256285548210144, 0.06923425197601318, 0.4820377230644226, -1.100419521331787, -0.01748935878276825, -0.2197563648223877, 0.9614702463150024, 0.27766159176826477, 0.4548223316669464, -0.6189822554588318, 0.542149007320404, -0.780157744884491, -0.3080541491508484, 0.9965453743934631, 2.030240297317505, 2.078810214996338, -0.43014222383499146, 0.5303628444671631, 0.4962296485900879, 0.2306182086467743, -0.2667608857154846, 0.18836243450641632, 1.8654098510742188, -0.7437435984611511, 0.22714415192604065, 0.3463789224624634, 0.31591206789016724, 0.8873323202133179, -0.32007020711898804, 1.6677210330963135, 0.4649519920349121, -1.2257510423660278, -0.2351962774991989, -2.3160998821258545, -0.40401020646095276, -0.0836135745048523, -0.7996001839637756, -0.42247825860977173], [0.07668716460466385, -0.5875176787376404, -0.444621205329895, 0.3109239339828491, -0.8887821435928345, -0.20881131291389465, 0.7802997827529907, 1.3554672002792358, -1.0801854133605957, -0.7467609643936157, -0.6723762154579163, -0.5496764779090881, 1.6473339796066284, 1.2088967561721802, -0.20852947235107422, 1.107346773147583, -1.3989008665084839, 0.34219497442245483, -0.07947255671024323, -0.5529568195343018, 2.5167109966278076, -1.683971643447876, 0.9614848494529724, 1.5200989246368408, 1.7867529392242432, 0.38236626982688904, 0.24300862848758698, -2.0835087299346924, -0.9340503811836243, -0.9348099827766418, 0.24028441309928894, -0.37541547417640686, -0.37330198287963867, -1.985573649406433, -0.4748256504535675, -0.05539720505475998, -0.7350600361824036, 0.32518914341926575, -0.6963399052619934, 0.5834032893180847, -0.0853554829955101, -1.235973834991455, -0.7658805251121521, -0.19078055024147034, -0.5321022868156433, 1.028290867805481, 1.9974254369735718, -0.8308994770050049, 1.439761996269226, 1.4956382513046265, -0.6760233640670776, -1.8483093976974487, -0.8150107264518738, 0.7990970611572266, 0.9188840985298157, 1.3502182960510254, 0.8903470635414124, -0.5241894125938416, 1.3127726316452026, 1.051618218421936, -0.1711946278810501, 0.6929891109466553, 1.3177095651626587, 0.09413327276706696], [1.1930474042892456, 0.47604888677597046, -0.5743017792701721, -0.3057563006877899, -0.15561266243457794, -0.6312997937202454, 0.923908531665802, 1.6247098445892334, -0.5802329182624817, 0.9491716623306274, -0.2734861969947815, 0.522633969783783, 1.9781705141067505, 0.9039233326911926, -0.47856399416923523, 0.5563782453536987, -1.1531414985656738, -0.15408679842948914, 0.9877866506576538, 0.8296210169792175, 0.11122935265302658, -0.5105071663856506, 0.024593716487288475, -0.8173584342002869, 0.28211578726768494, 1.1043591499328613, 0.327524870634079, -0.9859650731086731, 0.2465105801820755, 0.5679139494895935, -1.4385906457901, -0.08196261525154114, -1.3567136526107788, -0.8472339510917664, 1.1133095026016235, 0.4817815124988556, 1.5657695531845093, -1.514557957649231, -1.3876973390579224, 0.49123868346214294, 0.11522064357995987, 0.3770001530647278, 0.4642106294631958, -0.7399253845214844, 1.3932716846466064, 1.1601567268371582, -1.520810604095459, -2.383976459503174, -1.062867283821106, -0.5548760294914246, 0.2461278736591339, -0.5217881202697754, -0.17737968266010284, 0.7388865351676941, -2.004049777984619, -0.36465945839881897, -0.037399690598249435, -0.7892468571662903, 2.747260570526123, -0.041396331042051315, 0.45925024151802063, 0.10892469435930252, 0.6430565714836121, 0.2455817461013794], [-1.2429941892623901, 0.1306198388338089, 0.713125467300415, -0.8445116281509399, 0.060354843735694885, 0.07774019986391068, -1.4585087299346924, -0.6705752611160278, -1.331360936164856, -1.0362906455993652, 0.7474342584609985, 1.0129401683807373, -1.1324212551116943, 0.68767249584198, -1.333856225013733, 0.2042122185230255, -0.5091761350631714, 0.7732819318771362, 0.6984859108924866, 0.931753933429718, 2.419790267944336, 0.6468579769134521, 0.19707155227661133, 0.6469584107398987, 0.20857210457324982, 0.4094577133655548, -1.3680872917175293, 0.25710874795913696, -1.207879662513733, 1.1989914178848267, -3.4417967796325684, 0.598702609539032, -1.820067048072815, -0.07526848465204239, -0.2523481249809265, 1.319684624671936, 0.6334898471832275, -0.3933570683002472, -0.6851060390472412, -0.9370965957641602, -1.1507737636566162, 0.4819028973579407, -0.37863871455192566, 0.46839430928230286, -0.45651975274086, 0.2488262802362442, 0.5333894491195679, 0.789603590965271, -1.0037426948547363, -0.2802771031856537, -1.5903663635253906, -0.43155816197395325, 0.09815210849046707, -0.5721977353096008, -1.241018295288086, -0.6120377779006958, 1.0609385967254639, 1.109748363494873, -1.1382280588150024, 0.7810596227645874, 0.12643234431743622, -1.1105880737304688, 0.1298990100622177, 0.8667340874671936], [-0.5213276147842407, -2.6124842166900635, -1.2782905101776123, 1.2484158277511597, 0.7453453540802002, 0.2499586045742035, 1.6190053224563599, -0.6079224348068237, 1.5054177045822144, -0.6386011242866516, -1.1523960828781128, 1.0106260776519775, -0.7881340980529785, 0.6193816661834717, 1.2784634828567505, -1.5650744438171387, 1.8282966613769531, 1.2257449626922607, -0.1327342540025711, -1.5299030542373657, 0.07189176231622696, 0.252605140209198, 0.5554113388061523, 0.3826434314250946, -1.5288981199264526, -2.645658254623413, -1.237621545791626, 0.7566938400268555, -0.3929954767227173, -0.6267104744911194, 0.49773314595222473, -1.5634123086929321, 1.0417697429656982, 0.08896198123693466, -0.815033495426178, -1.1756751537322998, -0.6150657534599304, 1.3924245834350586, -1.345668911933899, -0.34263527393341064, 1.688263177871704, 0.319998174905777, 1.0106576681137085, -0.6002671718597412, 0.614367663860321, 1.3569424152374268, 0.5782439112663269, 1.1909862756729126, -0.45079195499420166, 1.2146865129470825, -0.3653108775615692, -1.6528631448745728, 0.1992364376783371, 2.056741952896118, 0.3297921419143677, 0.2967322766780853, -0.051299531012773514, -1.3321565389633179, 0.7979905009269714, 0.1018775925040245, -0.23587079346179962, 0.24938812851905823, -0.47804927825927734, -0.46726107597351074], [-0.04468843340873718, -0.19672374427318573, 0.42164260149002075, -0.27892622351646423, 0.4487324655056, -0.5905101895332336, -0.3525111675262451, -0.14649474620819092, -1.3690950870513916, 0.5447072982788086, 0.527641773223877, -1.423694133758545, -0.024808822199702263, 1.1973276138305664, 0.4300783574581146, 0.32452359795570374, -0.44809070229530334, 1.1752780675888062, 0.013392009772360325, 0.08407127112150192, 1.23688805103302, 0.9220257997512817, -0.6134756207466125, 0.6400721073150635, -0.4506889879703522, 0.7711969017982483, 0.659216046333313, 0.5658909678459167, -1.719672441482544, -2.71889328956604, -0.37607935070991516, 1.5918811559677124, 0.4997484087944031, -0.542113721370697, 0.8424434661865234, -0.008724487386643887, -0.34311652183532715, 0.8891430497169495, -0.9602020978927612, -0.6310967803001404, 1.6669657230377197, 0.5191088914871216, 0.454243540763855, 0.9714521169662476, -0.660865068435669, -0.8233457207679749, -0.5356746315956116, -0.04984460026025772, -1.872057318687439, -0.6982142925262451, 0.9943249225616455, 0.829139769077301, 1.5841493606567383, -0.6092256903648376, 1.1301730871200562, -0.5554614067077637, 0.5975469946861267, -0.8114904165267944, 0.8528610467910767, -0.8307222127914429, 0.7009144425392151, -0.4541250467300415, -0.5323483347892761, 2.048509359359741], [0.43854138255119324, -0.42238911986351013, -1.6316086053848267, -1.0770652294158936, 0.7483922839164734, -1.2146583795547485, 1.3327771425247192, -0.42183440923690796, 0.13062699139118195, 2.387106418609619, -1.8925529718399048, 1.8102809190750122, -0.30623385310173035, -0.11429060995578766, -0.275381475687027, 0.39273732900619507, 0.5072601437568665, 0.5885933637619019, -0.26783767342567444, -0.8400111794471741, -1.5288180112838745, 0.8931491374969482, 0.2949252426624298, 0.12858399748802185, -0.26749387383461, 0.03827761858701706, 0.31378886103630066, 1.1854099035263062, -0.8578926920890808, 1.6077226400375366, -0.9373394846916199, -0.8695709705352783, -1.1566704511642456, 0.5476629137992859, -0.8586040735244751, -1.0143184661865234, -1.2608646154403687, 0.6274732351303101, -0.9828140735626221, -0.5041677951812744, 0.04282154515385628, 0.5027757883071899, 0.2596970200538635, -0.3816792368888855, -0.7732969522476196, 0.29046958684921265, 1.408441424369812, -1.273206353187561, 0.09869879484176636, -0.8488057255744934, 0.4193703830242157, -0.4033485949039459, -1.3436427116394043, -0.06871320307254791, -0.5814003348350525, -0.20541974902153015, -0.6140182614326477, 1.1892058849334717, 1.5326300859451294, -0.7201412916183472, 0.18584506213665009, -2.149538516998291, 0.781547486782074, 0.24926888942718506], [0.028911156579852104, -0.9439051747322083, -0.0467144213616848, -0.40435343980789185, -0.253631055355072, -0.5787657499313354, 1.1502177715301514, 0.34052398800849915, -0.9187136292457581, -0.17665031552314758, -0.10649280995130539, 0.5029141306877136, 0.7082746624946594, -1.8608384132385254, 0.7856258749961853, -0.24620717763900757, -0.4452749490737915, 0.48851627111434937, -0.9579169750213623, 0.8490141034126282, -0.6778634786605835, -0.2686242461204529, -1.199053406715393, -1.5557684898376465, -0.5302826166152954, 0.3505336046218872, 0.8086658120155334, 0.5147157311439514, 1.2474662065505981, 0.21966390311717987, 0.25166448950767517, -0.32784587144851685, 0.5073608756065369, -2.424527645111084, 0.4260444641113281, -1.3342633247375488, 0.3701285123825073, 0.8342211246490479, -1.422666072845459, 0.273236483335495, -0.39250296354293823, 0.6767796277999878, -1.0040730237960815, 0.8424256443977356, -0.7083881497383118, -0.04457041993737221, 1.5994230508804321, 0.08752506971359253, -1.0923303365707397, 0.2819722592830658, 0.12663359940052032, -0.25850629806518555, 0.9548603892326355, -0.5264646410942078, -2.335667371749878, 0.08066362887620926, 1.076397180557251, 0.48646605014801025, -0.4772458076477051, -0.6737198233604431, -0.5740346312522888, -1.3031136989593506, 0.2459338903427124, 0.7379127740859985], [0.11721920222043991, 0.9377098679542542, -0.5317261815071106, -0.1249232068657875, -1.9151681661605835, -0.8898085951805115, -0.41600826382637024, 0.7382139563560486, -2.7564315795898438, -1.7297521829605103, -0.9101568460464478, -1.2277767658233643, 0.45765069127082825, 0.5193355679512024, 0.026903213933110237, -0.1858908236026764, -0.5505703091621399, 1.0022447109222412, -1.0299756526947021, 0.9463825821876526, 1.545060634613037, -0.5056449174880981, -0.8700310587882996, 1.5322424173355103, -0.8148247599601746, -2.191585063934326, 1.1282472610473633, -0.021985728293657303, 0.23727260529994965, 0.09454842656850815, -1.1637499332427979, 0.6301340460777283, -0.4614630341529846, -0.5960105061531067, -1.8711069822311401, -0.7109965085983276, 0.3600700795650482, -0.9661006331443787, -0.34267348051071167, 0.6086950302124023, -1.1716656684875488, -0.17454412579536438, 1.330352544784546, -0.6955015659332275, 0.7762933373451233, -0.6537707448005676, 0.7842280864715576, -0.6292468309402466, 0.3849782645702362, -1.2043110132217407, 1.2232545614242554, -0.5904677510261536, -0.4316226840019226, 0.04446455091238022, -0.9143532514572144, -0.3793427646160126, 1.6995683908462524, -1.676042914390564, -1.2865686416625977, -0.8200249075889587, -0.3784295618534088, -0.1427595317363739, -1.1878466606140137, -0.5161139369010925], [1.1854404211044312, -0.3264431953430176, 0.29756736755371094, -1.2875088453292847, 0.6672964692115784, 0.21038761734962463, 0.530633270740509, 0.34839096665382385, 1.5453922748565674, 1.5606276988983154, -0.1410239040851593, 0.4031648337841034, 0.819307267665863, -0.46313726902008057, -0.24332672357559204, 1.0500009059906006, -0.2943907678127289, -0.2535960376262665, -0.22140149772167206, 1.9882779121398926, 1.0968626737594604, 1.1396681070327759, 0.493912935256958, 1.119054913520813, 1.7759149074554443, 0.8917311429977417, -0.11723162233829498, -0.5094606876373291, -0.15532663464546204, -0.6684045195579529, -0.4745120704174042, 1.203186273574829, -0.47225379943847656, -1.2079347372055054, -0.804956316947937, 0.6501026153564453, -0.5486327409744263, 0.6897544264793396, -1.3091778755187988, 0.5370118618011475, 0.061236005276441574, 0.2843332588672638, -1.7437633275985718, 1.4450477361679077, 0.6134342551231384, -0.499850332736969, 1.8345662355422974, 1.0322622060775757, 0.8897027969360352, 0.7823497653007507, 0.12316332757472992, -0.1054249107837677, 0.405746728181839, 0.2827792167663574, 1.313500165939331, -0.03762947395443916, -0.47804051637649536, -0.7542642951011658, 0.03702778369188309, 0.5376543998718262, 0.7522211670875549, 1.021765947341919, -0.011361830867826939, -2.007366418838501], [0.0882779136300087, 2.0538716316223145, -1.3976601362228394, -0.7221515774726868, 0.30790117383003235, 0.8574022054672241, -0.6627349853515625, -0.585819661617279, 1.2424732446670532, -0.5403329730033875, -0.6981247067451477, 1.5819488763809204, 0.47412970662117004, -0.7322291731834412, 1.1310104131698608, -0.25536009669303894, -0.5139433741569519, -0.8439099788665771, 0.6521763801574707, -0.2153318226337433, -0.7595425844192505, -0.46130549907684326, 0.47090935707092285, -1.7017854452133179, -0.05459194257855415, 0.37192970514297485, 0.5119072794914246, 0.6362811923027039, 1.1456009149551392, -0.15211285650730133, 1.2928893566131592, 1.4056200981140137, -2.574399948120117, -1.450529932975769, -0.6126656532287598, 0.22563998401165009, 0.16863591969013214, -0.17509742081165314, -1.3534841537475586, 0.9707285165786743, -0.8125866651535034, -0.7216387391090393, 0.11337596923112869, 1.0038557052612305, 1.3386280536651611, -1.7581989765167236, -0.7102862000465393, -0.30085447430610657, -0.292400985956192, 0.40423572063446045, 1.7001683712005615, -1.1079388856887817, -0.7692712545394897, -2.1290283203125, 0.5108970403671265, 0.1926354169845581, -0.14676004648208618, 0.5223740935325623, 0.0799933597445488, 1.2527929544448853, -0.8918058276176453, -2.2449305057525635, -0.038463689386844635, 1.0728938579559326], [-0.9910711050033569, 0.27157482504844666, -0.07553870975971222, -0.7088860869407654, -0.6806311011314392, 0.7473884224891663, 0.5145153403282166, 1.3005613088607788, 0.5193532705307007, -0.5214716196060181, 1.3902957439422607, -0.9253705143928528, 0.9406365156173706, 0.024148255586624146, -0.6066660284996033, 1.0087677240371704, -1.5855728387832642, 0.682172954082489, -0.21861472725868225, -1.1288889646530151, -1.031441569328308, 1.5627250671386719, -0.5492950081825256, 0.4069605767726898, 0.7227702140808105, 1.056411623954773, -1.1598196029663086, 0.8457094430923462, -1.72519850730896, -2.1376659870147705, 0.4056817591190338, 0.8342352509498596, -0.8043886423110962, -1.3041226863861084, -1.111844539642334, -0.857408881187439, 0.3984925150871277, -0.3505919575691223, -0.7707293629646301, 0.37789949774742126, -1.4042388200759888, -0.09936054795980453, 2.2862164974212646, 0.16983312368392944, -0.5285690426826477, -0.48921069502830505, -0.20531786978244781, -0.03612624108791351, -0.8135346174240112, -1.5341776609420776, 1.1640366315841675, -0.9133990406990051, -0.1930910348892212, -2.0481553077697754, -0.2092600166797638, -1.5427135229110718, -0.49421125650405884, 0.12048129737377167, 0.250426322221756, -0.056950390338897705, 0.5825393795967102, 0.1655169129371643, 1.8833518028259277, 0.9551265835762024], [-2.3618357181549072, 1.0129209756851196, -1.7356988191604614, 0.5744614601135254, -0.8977518081665039, -1.2931932210922241, -1.5384941101074219, 0.09329050779342651, -0.47310349345207214, 0.8432874083518982, 0.5662622451782227, -0.4870911240577698, -1.2419075965881348, 0.6110861897468567, -0.02854873612523079, -0.04547898471355438, 0.49397605657577515, -1.1587622165679932, 1.0257774591445923, 0.28470662236213684, 0.25484782457351685, -0.29532700777053833, 0.5407467484474182, 0.19490306079387665, 0.5154479146003723, 1.2424827814102173, 0.7953919768333435, 1.2784801721572876, 1.396101713180542, 0.03862926736474037, -0.3989962637424469, 0.22392112016677856, 0.23462185263633728, 0.607947826385498, -0.8615962862968445, -1.4365862607955933, -2.2455685138702393, -1.147108793258667, -1.2904144525527954, -0.41980019211769104, -1.8317290544509888, -0.5303642749786377, 0.9716320037841797, 0.5277886986732483, 0.0034169345162808895, 0.7086110711097717, -1.626658320426941, -0.19769972562789917, 0.27553629875183105, -0.551042914390564, -0.943323016166687, -1.394715428352356, -0.5957043766975403, -1.8390185832977295, -1.41242516040802, -1.0981799364089966, 0.9694089889526367, 0.24106988310813904, -0.15491658449172974, -0.3316175937652588, -0.7938650846481323, 0.07957528531551361, 0.4098649024963379, -1.267564296722412], [0.46098119020462036, -0.4729520082473755, 0.5677224397659302, -0.15816456079483032, -0.2342168688774109, -0.24975526332855225, -0.9985324144363403, -0.486314982175827, 1.948395013809204, 0.5993806719779968, -2.370169162750244, -1.3857526779174805, 0.7303293943405151, 1.5741236209869385, -0.98548424243927, -0.3771805763244629, -2.4450302124023438, 0.800971269607544, -1.1176395416259766, -0.6751704812049866, -0.2807275652885437, -0.2924802005290985, 0.15712696313858032, 1.066163182258606, -0.45812365412712097, -0.20511141419410706, -0.14998815953731537, -0.5022998452186584, -0.2159442901611328, 0.7449694275856018, -1.4749581813812256, -0.2006349116563797, -1.0481986999511719, 0.9051037430763245, 1.3770538568496704, 0.5306805372238159, -0.08960002660751343, 0.33113378286361694, 0.10153818130493164, -0.3961956202983856, 0.9962816834449768, 0.5095541477203369, 0.9216373562812805, -0.7255234718322754, -0.2903219163417816, -1.1590372323989868, -0.8306787014007568, -2.001594305038452, -0.8394572138786316, 0.5067614316940308, -0.8709984421730042, 0.4942014217376709, 0.4625385105609894, -0.014348375611007214, 0.05179107189178467, -0.37138378620147705, 1.094202995300293, -0.6384931802749634, 1.263702392578125, 0.9379624724388123, -0.0850699320435524, -2.055004596710205, -0.3828750550746918, 0.023087425157427788], [1.5527931451797485, 0.26106059551239014, 0.6959978938102722, -1.118187665939331, -1.6805342435836792, 1.7409567832946777, 0.3338488042354584, -0.021984176710247993, -0.27958551049232483, -0.47917282581329346, -0.33882829546928406, 0.6041287183761597, -1.2962576150894165, -0.18534742295742035, 0.32472455501556396, 2.3395869731903076, -0.8243647217750549, -0.3676605820655823, 0.4462437927722931, 0.2530594766139984, 0.24032634496688843, -1.2725167274475098, -0.45334720611572266, -0.5499301552772522, -1.322387933731079, 0.44373589754104614, -0.8196164965629578, -0.5448039174079895, 1.4853001832962036, -1.3481181859970093, 0.6638485193252563, -0.09041658788919449, 0.628565788269043, -1.6106078624725342, -1.2361992597579956, 0.842557430267334, 1.2325609922409058, 0.4951644539833069, -0.6662788391113281, -0.7978770136833191, 2.2414093017578125, 1.3390401601791382, 0.9453212022781372, -1.1773345470428467, -0.026143167167901993, -0.3042057156562805, -2.007132053375244, 1.6780128479003906, 0.24060270190238953, 0.9762887358665466, -0.5292924642562866, -0.10270613431930542, 1.6403498649597168, -0.07862212508916855, -1.018955945968628, -0.31046682596206665, 0.6963955163955688, 0.34636348485946655, -0.46259763836860657, -0.8278608918190002, -0.442090779542923, -0.5208073854446411, 1.4796487092971802, -0.25220686197280884], [-0.6009188890457153, -0.31042802333831787, 1.7860641479492188, 0.9429245591163635, -0.4858885109424591, 0.15913541615009308, -1.0020639896392822, -1.3802196979522705, 0.17449824512004852, -0.9248477816581726, 2.264662027359009, -0.07639266550540924, -0.9017012119293213, -0.10947663336992264, 0.3955695629119873, 0.4768299460411072, -0.3059167265892029, 0.7872568964958191, -0.9032335877418518, -0.9827888011932373, 0.4634842574596405, 1.4747263193130493, 0.0012868725461885333, -0.2050548940896988, -0.6052971482276917, -1.59956955909729, 0.5464901924133301, 1.0327332019805908, -0.6721817255020142, 1.3755570650100708, 1.8977495431900024, -0.4754149615764618, -1.1756361722946167, -0.6295230984687805, 0.5643957257270813, 0.3372425436973572, 0.9382115602493286, -0.37435925006866455, -1.5446157455444336, -1.9885587692260742, -0.2588491439819336, 0.21365724503993988, -1.0009064674377441, -0.21401666104793549, 0.2000887542963028, 0.883039653301239, 1.4016785621643066, 0.517181932926178, -1.26862370967865, -0.8523356318473816, 1.269644021987915, -2.4543795585632324, 1.1774013042449951, 0.06629188358783722, -0.4980381727218628, -0.18240240216255188, 0.38687750697135925, 0.8705677390098572, 0.7159382700920105, -0.908654510974884, -0.1669471263885498, -0.3823722302913666, -2.5651400089263916, 0.05843402072787285], [0.5400933027267456, 0.16115634143352509, 1.2582104206085205, -0.9051364660263062, 0.08354710787534714, 0.9903565645217896, -0.8909186124801636, -1.1525534391403198, 0.0803660899400711, 0.3479709029197693, 1.101480484008789, -1.0010607242584229, 0.1850033402442932, -0.8637588024139404, 0.151280477643013, 0.4859704375267029, -0.6139229536056519, -1.2605674266815186, 1.4940822124481201, 0.5184389352798462, -0.3520047962665558, 2.048370122909546, 0.1577128916978836, 0.8702845573425293, -0.34697800874710083, 0.27730119228363037, -1.4294196367263794, -0.8992598652839661, 0.4433474540710449, -0.8598743677139282, -1.0053448677062988, -0.7095144987106323, 0.9399911761283875, -0.5948172807693481, -1.3630132675170898, -0.944063663482666, 1.6928976774215698, -0.3651803433895111, -0.7533702850341797, -0.8558085560798645, -0.4693782329559326, -1.5650928020477295, 1.1706898212432861, -0.6690353155136108, 0.16524897515773773, -0.6320568323135376, -1.0192461013793945, 0.8636200428009033, 1.7392256259918213, 0.20165367424488068, -1.1200436353683472, -0.14671500027179718, -1.683082938194275, -0.506951093673706, -1.5311522483825684, 1.3302667140960693, -2.2034175395965576, 1.2392699718475342, 0.7591361403465271, 0.11946562677621841, 0.08897396922111511, 0.9296711683273315, -1.2312617301940918, -0.11075513064861298], [1.0314205884933472, -0.39368557929992676, 0.6832237243652344, 1.014177680015564, -0.5085655450820923, -4.6290669441223145, -0.5549769997596741, -0.110970638692379, -0.43897223472595215, 0.060006700456142426, -0.8131662607192993, -1.2507914304733276, -0.46324974298477173, -1.9624614715576172, 0.11520520597696304, 0.30300799012184143, -0.5636764764785767, -1.5523377656936646, -1.7347368001937866, -1.9971257448196411, -0.18974420428276062, -0.2711848020553589, -1.3098000288009644, 0.18427595496177673, -0.6867564916610718, 0.17437593638896942, -1.9857741594314575, -0.14600269496440887, 0.3741839528083801, -0.855395495891571, 0.15424484014511108, 0.8229100108146667, -0.06885023415088654, -0.3780301809310913, 0.0912708193063736, -0.29663464426994324, -0.33324572443962097, -1.4699751138687134, -0.1332835704088211, -0.356355220079422, -1.475661277770996, 1.4391757249832153, 0.7695416808128357, 0.6521621942520142, -0.11534405499696732, -0.20394834876060486, 0.6801326274871826, 1.2911909818649292, -0.21854816377162933, -0.4519517123699188, -0.7247039675712585, -1.4357401132583618, 1.3041627407073975, -1.4011858701705933, -0.265392541885376, 0.34520241618156433, 0.09154678881168365, 0.3115774989128113, -1.0314701795578003, -0.0891982838511467, -0.7598198056221008, 1.289658784866333, 1.3439444303512573, -0.18168538808822632], [-0.4012431502342224, 0.0859561637043953, 0.844920814037323, 0.2781025767326355, 0.1080341786146164, -0.4333662986755371, 0.6501303911209106, 1.2556259632110596, -1.2018135786056519, -0.23166996240615845, 0.7370079159736633, -1.2273447513580322, -1.0045974254608154, -0.5628888010978699, 0.4670233130455017, 0.33674442768096924, 1.0693659782409668, -0.9340548515319824, 0.7277906537055969, 1.57620370388031, 0.4319700002670288, 0.2749897241592407, -2.327972173690796, 0.9476677775382996, 0.9983237385749817, 0.38226819038391113, 0.31776168942451477, 0.15488201379776, -1.0488561391830444, -0.06785038858652115, -0.7394896745681763, -0.6245318651199341, 1.1264331340789795, -0.7207741141319275, -0.2933714985847473, -0.004779884126037359, -0.6653853058815002, 0.3904447853565216, 0.07610482722520828, 1.0320037603378296, 0.1151803508400917, -0.02405141107738018, 0.32765620946884155, -1.3178311586380005, -2.0092127323150635, -0.4447283446788788, -0.677897572517395, -0.044698458164930344, 0.33369848132133484, -0.1397739201784134, -0.022108251228928566, -1.1166647672653198, 2.131082057952881, 0.038615550845861435, -1.620362401008606, -0.662921667098999, -1.070678472518921, -0.29269206523895264, 1.349759578704834, 0.22370441257953644, -0.467891663312912, 0.11478099226951599, -2.0678064823150635, -0.4635936915874481], [1.1569784879684448, 0.9532164335250854, 1.6716746091842651, 0.14201749861240387, -0.9966297745704651, -0.9885920286178589, 0.038566481322050095, 0.2660360038280487, 1.178666591644287, -1.5689918994903564, -0.6277214884757996, 0.19933465123176575, -1.3638153076171875, 1.4057344198226929, 0.36735036969184875, -0.5124772787094116, -2.3153727054595947, 0.8098348379135132, -0.4356733560562134, -0.33675429224967957, -0.775979220867157, 0.9980608820915222, 0.1979834884405136, -2.2135775089263916, 1.4386218786239624, 0.17159752547740936, -0.9687996506690979, -0.8626218438148499, 1.8242473602294922, 1.0385254621505737, -1.2828775644302368, -0.8104584217071533, -0.4818996489048004, 0.5003370046615601, -0.08988091349601746, 1.3747292757034302, -0.479146271944046, -2.170170545578003, -0.15833979845046997, 0.690260648727417, 1.5657984018325806, 0.962315022945404, 0.1457691788673401, 0.664050817489624, 1.4028414487838745, -0.06742715835571289, -2.3446247577667236, 0.7210273742675781, 0.01868910901248455, -0.6197746992111206, -1.0019479990005493, 0.20433183014392853, 0.8583318591117859, 0.9648444652557373, -0.1328336000442505, 0.46094220876693726, 0.720601499080658, -1.7418521642684937, -1.4189516305923462, 0.33811724185943604, 0.36525532603263855, 0.24004019796848297, 0.636104941368103, 0.4965221583843231], [-1.932814121246338, 0.49692919850349426, 1.1465907096862793, -0.9099805951118469, 0.17111168801784515, 1.7394777536392212, 1.3724586963653564, 0.2571158707141876, -1.1711933612823486, 0.5986082553863525, -1.1846002340316772, -1.742303729057312, 1.5774160623550415, -1.1882215738296509, 0.6659202575683594, 0.901215672492981, 1.2138737440109253, -0.7715428471565247, -0.8265814781188965, -0.17818686366081238, -0.9376250505447388, 0.9017367362976074, 1.245895266532898, 1.2130422592163086, -1.0233205556869507, 0.06903491914272308, -0.5127844214439392, 0.16275812685489655, 0.6929664015769958, -0.9522714614868164, 0.16961483657360077, 0.9137532711029053, 0.7359247207641602, -1.1732361316680908, -0.29686176776885986, 1.7563984394073486, 0.19938941299915314, 0.15232928097248077, 0.6207601428031921, -0.15641234815120697, 0.5047426819801331, -0.34701403975486755, 0.6710361838340759, 1.5641381740570068, -2.835308313369751, 1.2490206956863403, -0.4114600121974945, 0.45865291357040405, 2.2410571575164795, -1.1950372457504272, 1.9625134468078613, -1.2328362464904785, -0.4127088487148285, -1.0673996210098267, -0.24115844070911407, 0.7360127568244934, -0.8162620067596436, 0.10618942975997925, -0.7933017015457153, 2.482167959213257, 0.4846615195274353, -0.6403110027313232, 1.5145576000213623, 0.3466937243938446], [-1.1092528104782104, -0.37283843755722046, 0.2673525810241699, -1.387110948562622, 0.27927905321121216, 2.773076057434082, -0.7053730487823486, 0.5446247458457947, -0.8569129109382629, -0.3113301992416382, -0.2301098108291626, -2.61858868598938, 0.4116848409175873, 0.33041301369667053, 0.5906946659088135, -1.5239784717559814, -1.556504249572754, 0.9469809532165527, -1.1986985206604004, 1.197062611579895, 0.19095125794410706, 1.0459482669830322, 2.1012794971466064, 0.2700416147708893, -0.9099200367927551, 0.3100862503051758, -1.0080307722091675, 0.8555020689964294, -1.170220971107483, -0.3529738783836365, -0.14380629360675812, -0.892826497554779, 1.293880581855774, 0.3015884459018707, 0.17266035079956055, -0.02938898652791977, 0.05838571488857269, -1.3548195362091064, -0.8533521890640259, -0.3982574939727783, 0.5318982005119324, -0.9782785773277283, -0.07353177666664124, -1.0633543729782104, 0.32286742329597473, -1.8089054822921753, -0.15741051733493805, 0.81587153673172, -1.11299729347229, 0.33795249462127686, -0.5512040257453918, 0.764471709728241, -1.6306930780410767, 0.058017220348119736, 1.4302411079406738, -1.428930640220642, 0.0860595852136612, -0.7378701567649841, -0.24755166471004486, -0.8110476732254028, -0.5558957457542419, 1.3546967506408691, 1.0953775644302368, -0.019745340570807457], [0.014973997138440609, 0.5677705407142639, 0.9965234994888306, -0.4705486595630646, 0.28704833984375, 1.3667936325073242, 1.82472825050354, -1.5428036451339722, -0.3453693091869354, -0.11072627454996109, -0.2656800150871277, 1.3155622482299805, 1.1058868169784546, 0.311021089553833, -0.6285591721534729, -2.103191375732422, 0.024741873145103455, 1.038400411605835, -0.06409208476543427, 1.0094815492630005, 0.39162877202033997, 0.8720937967300415, 1.9446425437927246, 0.5635897517204285, 0.6812009215354919, -0.2538948655128479, -0.6314706802368164, -0.8064656853675842, -0.41728475689888, -0.6821759343147278, -0.11569710075855255, -0.3145287036895752, 0.7883261442184448, -1.1319353580474854, 0.4253719747066498, 0.9771663546562195, 0.38620659708976746, 0.02610214240849018, -1.147708535194397, 0.9756163954734802, -0.438488245010376, -0.9803982973098755, -1.7698643207550049, -1.134860634803772, -0.7999606728553772, -0.185594379901886, 0.7906946539878845, -0.7948629856109619, -0.1548076868057251, 0.19679376482963562, 0.40115174651145935, 0.0001562568504596129, 0.19047491252422333, -0.37786808609962463, 1.0034610033035278, 1.307219386100769, -1.6597059965133667, -0.6533241868019104, 0.3262743353843689, -0.11669570207595825, 0.9951409101486206, -0.9202388525009155, 0.906438946723938, -1.9868645668029785], [-1.1312354803085327, -1.4146521091461182, -0.3098193407058716, 0.9617881178855896, 1.3690097332000732, -0.19593779742717743, 1.3903250694274902, -1.8755747079849243, 0.22214603424072266, -0.21441851556301117, -0.3748849332332611, -2.1607723236083984, -0.3790128529071808, -1.1301859617233276, 0.28677159547805786, 0.8860346078872681, 0.7235311269760132, -0.528194010257721, -0.6470099091529846, -0.057091593742370605, 0.8155428171157837, -0.22324705123901367, 1.3710103034973145, -0.3123268485069275, -1.2154921293258667, 1.6965196132659912, -0.09099514782428741, -0.06787995994091034, -0.011296957731246948, 0.5754408836364746, -0.7050235867500305, -1.6535060405731201, 0.5762226581573486, 0.39906546473503113, -1.840077519416809, 2.835179567337036, 0.6337624788284302, -1.7199474573135376, -0.3388705253601074, -0.29191288352012634, 0.25161296129226685, 0.41107645630836487, -0.35848841071128845, -1.3135215044021606, 0.2473912239074707, 0.48118487000465393, 0.28179487586021423, -0.3278163969516754, 0.3568568825721741, 1.2508747577667236, 0.5131089091300964, 2.1169729232788086, -0.27104899287223816, 0.4008733034133911, 0.23472394049167633, -1.6755295991897583, 0.18079181015491486, -0.4849644899368286, -0.42416995763778687, -1.7079570293426514, -0.27064839005470276, -0.584448516368866, 0.7950566411018372, 0.4078447222709656], [-1.834626317024231, 0.3129960298538208, 0.7781229615211487, -0.977636992931366, 1.049696683883667, 1.9210175275802612, 0.5060784220695496, 0.027290094643831253, 1.996846079826355, 1.759588360786438, 0.9243980646133423, 0.8478417992591858, -0.19585749506950378, -0.44797345995903015, 0.024952160194516182, -0.6433039903640747, 0.5044451951980591, -0.47759097814559937, -0.17944861948490143, -0.15650659799575806, -1.4036978483200073, 0.2349199801683426, -0.3299316465854645, 1.3631805181503296, 0.30631211400032043, -0.6398053169250488, -0.19795264303684235, -1.2526112794876099, 1.1816867589950562, 0.20950782299041748, -0.02886693738400936, -0.08734051883220673, 1.0677348375320435, 0.08011815696954727, 0.5889947414398193, 0.5382658243179321, 1.2867205142974854, -1.4777030944824219, -0.36632513999938965, -0.37319204211235046, -0.3564576506614685, 1.615111231803894, 1.3081591129302979, -0.15256178379058838, -1.4341871738433838, -0.3375328481197357, -0.5322774052619934, 0.14626620709896088, 1.396508812904358, -0.16065017879009247, -1.1991961002349854, 0.6206809878349304, 1.729830265045166, 1.0680795907974243, -1.210625410079956, -0.7288339138031006, 0.02361864224076271, -0.5053084492683411, 0.48478397727012634, 1.0997281074523926, -0.9628013372421265, 0.8347424864768982, -0.6177055835723877, 0.6632609367370605], [1.3179210424423218, 0.922551155090332, 0.003276602830737829, -0.3519473075866699, 0.2842263877391815, -0.7015284299850464, -0.5656956434249878, -1.1090070009231567, -0.570743203163147, 0.1551356464624405, -0.701365053653717, -0.047480616718530655, 0.8280162811279297, -1.165993094444275, 0.3503647446632385, -0.567497968673706, -0.312095046043396, 0.5982881784439087, -0.38165009021759033, 0.94320148229599, -1.7337454557418823, 1.8149610757827759, -0.601168692111969, 0.7707837224006653, 0.46984803676605225, -0.05219846963882446, -0.8105849027633667, -1.9891560077667236, -1.3288798332214355, -0.8428810834884644, -0.7182584404945374, 0.5543776154518127, -0.5952656865119934, -0.0004220919217914343, -0.6005561947822571, 0.24623073637485504, 0.24444638192653656, 1.772110104560852, 0.02110057882964611, -0.4225279688835144, -0.9522049427032471, 0.1478908509016037, 0.3350379168987274, 0.11631010472774506, 0.1729881465435028, 1.8857293128967285, 1.2748947143554688, 0.28689342737197876, 0.6434539556503296, -0.2492770403623581, 0.2073701024055481, -1.6916548013687134, 0.9163685441017151, 1.4920786619186401, -0.3230395019054413, 0.2793174386024475, 0.1247544139623642, 1.8727102279663086, -0.4409906566143036, 0.9345269203186035, -0.21614611148834229, 1.216370701789856, -0.8682871460914612, 0.642020583152771], [-0.4760937988758087, 0.6032645106315613, -0.9064381718635559, 0.7881042957305908, -1.2546749114990234, -1.2726752758026123, -0.6774206757545471, -0.3051517605781555, 0.09787368774414062, 1.1044220924377441, -1.2704541683197021, 0.7658719420433044, 0.7026405930519104, 1.578627109527588, 1.174686074256897, -0.6500259637832642, 0.8404528498649597, 1.0809627771377563, 1.700374722480774, -1.295684576034546, -0.010398024693131447, -0.49893611669540405, -1.4018255472183228, 1.1306564807891846, -0.6175745129585266, -0.7330567836761475, -0.5002708435058594, -0.7297372221946716, -0.3867102563381195, -0.5761665105819702, 2.8786733150482178, -0.27415651082992554, -0.1478499174118042, 0.9709003567695618, -0.4143623411655426, 1.3447080850601196, -1.0484358072280884, -2.4617011547088623, 0.3700597286224365, -0.40016841888427734, 0.6781448721885681, 0.17140749096870422, -0.34150996804237366, 0.2974106967449188, -1.0780147314071655, -1.2371102571487427, -1.1299684047698975, 0.32113948464393616, 0.15913204848766327, -0.4901584982872009, 0.8963325023651123, -0.1836937814950943, 0.11274909973144531, -0.4497215449810028, -1.0754741430282593, 1.5051405429840088, -0.5246280431747437, 0.2963380217552185, 0.8548487424850464, -0.3473595082759857, 1.043874979019165, -0.0029783633071929216, -1.2495695352554321, -1.5016205310821533], [-0.6557880640029907, -1.3307294845581055, -1.6314302682876587, 0.9889432191848755, 1.2354421615600586, 2.168344497680664, -1.8382865190505981, 2.9165143966674805, 1.4011555910110474, 0.1785867065191269, -0.3475601375102997, -0.7967731952667236, -1.5628191232681274, -0.30004581809043884, -0.3318839371204376, 1.1269514560699463, 1.1804438829421997, -0.2183164358139038, 0.5841485857963562, 0.7202145457267761, 0.37690746784210205, 0.9656908512115479, -1.8866530656814575, -0.0643836110830307, 1.200351357460022, -0.3221816420555115, -0.584007203578949, 0.2867792546749115, 0.3146541118621826, 1.8654934167861938, 1.448452353477478, 0.565219521522522, -1.7670775651931763, -1.3357611894607544, 0.04598471149802208, 3.315436363220215, 0.4986650347709656, -0.45741918683052063, 2.1240200996398926, 1.3340305089950562, 1.5905461311340332, -0.5044564604759216, -0.5644392967224121, 1.6495115756988525, 0.6267001032829285, 0.2566976845264435, 0.5748804807662964, -0.8750669360160828, 0.7556220889091492, -1.3853586912155151, 1.8391547203063965, -0.09463001042604446, 0.742195188999176, 0.7194712162017822, -0.2498829960823059, -1.1320818662643433, 0.21302486956119537, 0.7353973388671875, 0.8665972948074341, -0.7502791881561279, -0.5171537399291992, -0.11932946741580963, 0.14872623980045319, 0.040916021913290024], [-1.2322982549667358, 0.1361144483089447, 0.652496337890625, -0.7752754092216492, 1.1638829708099365, -0.056272730231285095, 0.8011202812194824, 0.661900520324707, -0.5689342021942139, -0.23555663228034973, -1.2425544261932373, 0.42471030354499817, -0.5889633297920227, -0.803753674030304, 0.2696438431739807, -1.4443950653076172, 0.6481082439422607, 1.5853605270385742, 0.6819706559181213, -2.940957546234131, 0.17118901014328003, 0.9219520092010498, 0.7878134250640869, -1.7036867141723633, 0.7898266911506653, -0.39920273423194885, 0.7995916604995728, -0.3994353413581848, 0.8905231356620789, -1.0315457582473755, 0.7468706369400024, 0.5586966276168823, 0.8740423917770386, -0.13209322094917297, -0.6954005360603333, 0.33242785930633545, -0.7651333212852478, 0.9515025019645691, 1.384122610092163, -1.4724931716918945, 0.4011688232421875, -0.9224166870117188, -1.34170401096344, -0.766313910484314, 1.2554502487182617, 1.1148918867111206, 0.675737202167511, 0.6527566313743591, 1.7397984266281128, -0.41073668003082275, 1.1743263006210327, 1.7690223455429077, -1.953741192817688, -0.696308434009552, -1.7512990236282349, -0.7536231875419617, -0.8792287111282349, 0.6889670491218567, -0.28313493728637695, 0.41904908418655396, -1.734458327293396, -1.2399171590805054, 1.2945878505706787, 0.2240479588508606], [-0.5170770883560181, -2.38433575630188, 1.6482661962509155, -0.4969322085380554, 2.1399738788604736, 0.5424112677574158, -2.1105942726135254, 0.588618814945221, -0.6270832419395447, 0.6153729557991028, -1.513881802558899, -1.2113523483276367, -0.12962861359119415, 0.34702253341674805, 2.134018898010254, 1.7226635217666626, 0.7286088466644287, -0.7470564842224121, 1.6253656148910522, 0.10250777751207352, 0.11891350150108337, -0.682567298412323, 0.10548838973045349, -0.05913977324962616, 0.44781816005706787, 1.029248595237732, -0.022895853966474533, 0.5749671459197998, -0.46258291602134705, -0.5147716999053955, 1.890681505203247, 0.3990509510040283, 0.36981523036956787, 0.8287100195884705, -0.026915444061160088, -1.7507634162902832, 0.9463449716567993, 0.9716393351554871, 0.06058765575289726, -0.2891826331615448, 0.089370496571064, 0.470647931098938, -1.5347576141357422, 1.8347313404083252, -0.8085732460021973, -0.6418586373329163, 1.7369142770767212, -1.3462680578231812, 0.30913564562797546, 0.1617622971534729, -0.8637709617614746, -1.014591932296753, 0.08491335809230804, 1.9692373275756836, 1.1365280151367188, -0.9998248815536499, 1.1966955661773682, -0.6487292647361755, -0.07746293395757675, 1.5603199005126953, 1.1731783151626587, 0.23751994967460632, 1.088078260421753, 0.47611725330352783], [-0.8174632787704468, 0.6040864586830139, 0.8976229429244995, -0.83925861120224, -1.1609563827514648, 0.09705713391304016, 1.0941925048828125, 0.647885799407959, -0.06179587543010712, -1.2038911581039429, -1.131134033203125, -1.5146989822387695, -1.333642601966858, -1.4544343948364258, -0.10027840733528137, -2.482172966003418, 0.848041832447052, 0.20156174898147583, 0.6387426257133484, -0.27694177627563477, -0.3104490637779236, 0.636538028717041, -0.13220995664596558, -0.716525673866272, 0.25034359097480774, 0.9224073886871338, 1.1608490943908691, -0.8650882244110107, 0.6806667447090149, 1.3922268152236938, 0.03986199572682381, 0.20995505154132843, 0.3725299835205078, -0.891057550907135, 0.8093224167823792, 0.7968326807022095, 0.20341122150421143, -1.2913340330123901, -0.17927895486354828, 0.15880802273750305, -1.3037641048431396, -2.172001600265503, -0.22987066209316254, 0.31964612007141113, 0.3234979808330536, 0.143724262714386, 0.3515395224094391, -0.45400628447532654, 0.4471069872379303, 0.8351234793663025, 1.6692713499069214, 0.17004486918449402, -1.3383429050445557, 0.4289117157459259, -1.6259698867797852, 1.8537752628326416, -1.024531602859497, 2.018407106399536, -1.770520806312561, 1.2757679224014282, 2.067930221557617, -0.3552546799182892, 0.6885693669319153, 0.32227790355682373], [0.49540191888809204, -1.2918106317520142, 0.47527801990509033, -0.6321828961372375, 0.2651882469654083, 1.2857763767242432, -1.04874587059021, -0.3734321594238281, -0.773661732673645, 0.5916727185249329, -1.0782158374786377, -0.9225055575370789, -0.38759398460388184, 0.280777245759964, 0.453061580657959, 0.6307446956634521, -0.5378268957138062, 1.1231300830841064, 0.5118219256401062, 0.532524049282074, -0.02359018661081791, 0.6348083019256592, 0.5397751331329346, -0.48653608560562134, 0.14809614419937134, 0.7564443349838257, -1.208071231842041, -1.4345998764038086, -0.9822033047676086, -1.3226155042648315, -0.9465600848197937, -1.8512458801269531, 0.18858137726783752, -0.3023315966129303, -0.5863597393035889, -0.09608110785484314, -0.894376814365387, -0.8540116548538208, 1.0167697668075562, 0.6171267032623291, -1.4461348056793213, 0.5400562882423401, -0.4892636239528656, -0.21945863962173462, -0.6432626843452454, -0.5173233151435852, -0.09879887849092484, 0.7141818404197693, 0.16703909635543823, -1.6026371717453003, -0.20146091282367706, -0.669783353805542, 1.2363752126693726, -1.7985424995422363, -0.9108516573905945, 0.023321351036429405, -1.165696620941162, 1.6210068464279175, -1.4499417543411255, -0.0903313010931015, 0.37262052297592163, 0.5689749717712402, -0.10984691977500916, 0.6492645740509033], [0.5105515122413635, -2.721891403198242, 1.6034512519836426, 1.518358826637268, 1.7444407939910889, -0.24558544158935547, 0.25380557775497437, 0.44456544518470764, -1.3074792623519897, -0.5277948975563049, -2.417605400085449, -1.3004509210586548, 1.8860669136047363, -0.2803829312324524, 0.8608052730560303, 0.4484303891658783, 0.710890531539917, 0.3162704110145569, 0.08088602870702744, -0.2228451669216156, 2.2532949447631836, 0.3709743618965149, 0.18487393856048584, 0.5927146673202515, 0.1644650101661682, 1.2171679735183716, 0.26444774866104126, -0.6747443675994873, -1.0851335525512695, -1.5741454362869263, -0.41965967416763306, -1.5115736722946167, 0.11649910360574722, 0.26107057929039, 2.547435998916626, -1.289007544517517, -0.38458263874053955, -0.03766655549407005, -1.0471017360687256, -1.5294270515441895, -2.011361598968506, 1.4737389087677002, -0.7221159934997559, -1.8892529010772705, 0.07551979273557663, -0.9255204796791077, 0.04490650072693825, 0.4444803297519684, 0.27453067898750305, 0.314236581325531, 1.9788049459457397, -1.9545270204544067, 0.12385478615760803, 1.506375789642334, -1.4133635759353638, -0.87149977684021, -0.8387166261672974, -0.2237972617149353, -1.5453991889953613, 0.2509785592556, -1.1474806070327759, -0.8669037818908691, 0.28540265560150146, 1.7434749603271484], [0.26220768690109253, -2.608844757080078, -0.7555274963378906, -1.086209774017334, 1.528128743171692, -0.7952926158905029, 1.4225451946258545, -0.2697085440158844, 0.7720780968666077, 1.6869351863861084, 0.15139548480510712, 0.08613565564155579, -1.5269891023635864, 0.9419683814048767, 0.7085294723510742, 1.4400007724761963, 1.4851452112197876, 0.25540971755981445, -1.1027679443359375, -0.054023925215005875, -0.5983564853668213, -0.14886906743049622, -1.1822048425674438, -0.20850896835327148, -0.2396455854177475, -0.09494627267122269, -0.6238245964050293, -0.2604309916496277, -0.192446768283844, -0.3095361590385437, -0.27660349011421204, 1.4136881828308105, 0.7331141829490662, -1.1711626052856445, 0.6347652077674866, -1.1542378664016724, 1.4060876369476318, 0.39245375990867615, 0.3020535409450531, 1.5262672901153564, 0.27371275424957275, 0.29167667031288147, -0.01378659252077341, -0.9425381422042847, -1.5072057247161865, -1.943493366241455, 0.9021303653717041, 0.36251845955848694, -0.4807094633579254, 1.0951809883117676, 0.19274160265922546, -0.22173593938350677, -0.9025784134864807, -1.0240237712860107, -1.805212378501892, -0.3294433057308197, -1.4386399984359741, 0.859453558921814, 0.13191355764865875, -0.9847921133041382, -1.0907628536224365, -0.7524983882904053, 0.170548215508461, 0.5637462735176086], [0.45666974782943726, 0.9610538482666016, 0.9159356951713562, 1.36976957321167, -1.840167760848999, 0.33860549330711365, -1.1363903284072876, 1.7792376279830933, -1.4166675806045532, -0.2767266631126404, 1.8305033445358276, 0.8540837168693542, -0.8549237251281738, -0.09558362513780594, 0.3276509940624237, 0.36807766556739807, -1.6388081312179565, 1.42360258102417, -0.5778306126594543, 0.27919402718544006, 2.1084141731262207, 0.7614844441413879, -0.6286227703094482, -0.8746630549430847, -0.22284220159053802, -0.464618980884552, -1.526959776878357, 1.3029179573059082, -1.9919990301132202, 0.3121298551559448, -0.6681156754493713, -1.9191694259643555, -0.6365941762924194, -1.112888216972351, -0.10404820740222931, -1.2310235500335693, -1.296531319618225, 1.3174856901168823, 0.7626345157623291, -0.04326757416129112, 0.8766117095947266, -1.2207813262939453, 0.9437196254730225, -0.7370832562446594, -0.3275403380393982, 0.8241754174232483, -0.32829535007476807, -0.4000502824783325, 0.7123207449913025, 1.1679621934890747, -0.07329367101192474, 0.4348123371601105, 0.3524009883403778, -0.4710310101509094, -0.2054538130760193, -1.1013813018798828, -1.178185224533081, -1.1655596494674683, 0.057814519852399826, 0.01161694061011076, 0.2843736410140991, 1.9298104047775269, -0.6424618363380432, -0.4746108949184418], [1.5417180061340332, -0.36716198921203613, 0.095561183989048, -0.978113055229187, -1.2032593488693237, 0.9813849329948425, 0.20074854791164398, 0.6519567370414734, 1.4311654567718506, -1.124468207359314, -2.034532308578491, -0.5507949590682983, 0.10779315233230591, -0.011006628163158894, 1.5315896272659302, -0.019219176843762398, -2.0557775497436523, 0.983041524887085, 0.9209614992141724, -0.5768166184425354, 0.6089483499526978, -0.5016067624092102, -1.8685622215270996, 1.6905958652496338, 0.11932235956192017, -0.22599318623542786, -1.3236428499221802, 0.3184342682361603, -0.9724565744400024, -0.17898696660995483, 2.5552585124969482, 0.024868261069059372, -0.5444366335868835, 2.1555075645446777, -0.7896118760108948, -2.0227813720703125, 0.566981852054596, -0.07541761547327042, -0.3465490937232971, 1.0578941106796265, -0.5197979807853699, 1.1458475589752197, 0.6404662132263184, -0.9317067265510559, -0.7985338568687439, 0.4186687767505646, -1.364182710647583, -0.5693333148956299, -0.16979031264781952, -0.8617865443229675, -0.12431642413139343, 0.5969573855400085, 0.25179609656333923, -0.3315831422805786, 2.216189384460449, -0.47106266021728516, -0.9459871053695679, 1.5175533294677734, -1.0499118566513062, -0.8880124092102051, 2.233511447906494, -0.8092971444129944, -0.8516100645065308, -2.4802021980285645], [-1.5808817148208618, -0.72252357006073, 0.7101666927337646, 1.0978453159332275, -0.9216936230659485, -3.171168804168701, -1.1306788921356201, 0.8515297174453735, -1.021882176399231, -0.6531396508216858, -0.13159845769405365, 1.7199259996414185, 0.9661201238632202, -0.0054761325009167194, -0.04832955822348595, -0.3316297233104706, -0.4273667633533478, -0.9583332538604736, -0.8517497777938843, 0.2468772828578949, -0.4009363353252411, -0.5930611491203308, 0.2688865065574646, -0.5511480569839478, -0.6300588250160217, 0.4394380450248718, 0.9870654940605164, -1.367371916770935, 0.1283254772424698, -0.630480945110321, 0.030397402122616768, 1.3328437805175781, -0.6913304328918457, 0.28848597407341003, -1.4954346418380737, 0.599545955657959, -0.2559206187725067, 0.16902194917201996, -0.1678154021501541, -1.7037034034729004, 0.48780685663223267, 0.5910115242004395, 0.058353833854198456, 0.2464328557252884, 0.5248867869377136, -0.04705410078167915, -1.5807057619094849, -0.18967072665691376, 0.529278039932251, 0.2006838470697403, -0.09142225235700607, 0.2689531445503235, 0.38505539298057556, -1.7408344745635986, 1.1791969537734985, -0.18052886426448822, 0.20186714828014374, -0.22877031564712524, -0.41808241605758667, -0.3192461431026459, -0.40203508734703064, 0.6488448977470398, 0.3358732759952545, -0.11336783319711685], [0.9110917448997498, 2.2751400470733643, -1.136088252067566, -2.1501739025115967, -1.0020389556884766, -0.6036624312400818, 1.0421396493911743, -0.501947283744812, -0.8121955394744873, 0.1551545113325119, -1.8521387577056885, 0.3551020324230194, 1.0389339923858643, 0.6784165501594543, -0.7617575526237488, 0.19094379246234894, 0.5919559597969055, -1.9486690759658813, 1.3749890327453613, -0.16127555072307587, -0.6084434390068054, 0.8451439142227173, -0.9961170554161072, -1.2092742919921875, -0.5337964296340942, 0.6944295167922974, 1.1229453086853027, -0.4312608540058136, -1.7101576328277588, -0.530598521232605, -1.70858633518219, 0.3745669722557068, -1.4806780815124512, -0.6822846531867981, 1.9613213539123535, 0.4980984032154083, -0.3468032479286194, 0.2935287356376648, 1.6447607278823853, -0.5134201049804688, -0.4274449646472931, 1.3640656471252441, 0.1235160157084465, -0.6913281083106995, 0.5118944048881531, -0.44384798407554626, -1.8402827978134155, 0.2743368148803711, 0.09561115503311157, -0.14622309803962708, 1.548563838005066, -1.2028312683105469, 1.1188284158706665, 0.49673527479171753, -0.8515403270721436, 0.4765722155570984, -0.29502061009407043, -0.19291909039020538, -1.4756577014923096, 1.1252459287643433, -1.9238965511322021, -0.9134806394577026, 0.7833312153816223, 0.3913320302963257], [-0.09527799487113953, 0.22121351957321167, -0.28509628772735596, -0.10071315616369247, 0.18436962366104126, 0.7244819402694702, 0.3396850824356079, 0.9463155269622803, 0.6452867984771729, 0.1872565597295761, 0.8722094893455505, -0.39451977610588074, 0.5383416414260864, -0.7245604991912842, 1.233801245689392, -0.8624593019485474, 1.4639497995376587, -0.15020009875297546, -2.0713746547698975, -0.3784385323524475, 1.4780495166778564, 1.665579080581665, 1.9766132831573486, 0.1091151162981987, 0.18334336578845978, -0.7838149070739746, 1.1937024593353271, -0.2801913321018219, -0.7414352893829346, 0.08283614367246628, -0.6150899529457092, -0.9623590111732483, -0.9631456136703491, -0.3267495930194855, -1.3168524503707886, 1.8958629369735718, 1.5943626165390015, -0.4014425277709961, -0.2119094729423523, -0.23192210495471954, -2.0548641681671143, 1.2396056652069092, 0.19288724660873413, -1.8060113191604614, -1.3221906423568726, -1.2209824323654175, -0.5497888922691345, -0.43468740582466125, 0.1994887739419937, 0.40433481335639954, 0.034797538071870804, -0.6226457357406616, 1.8011304140090942, 0.0448637418448925, -0.9723323583602905, -2.2265095710754395, 1.5463101863861084, 1.0459296703338623, 0.07880830764770508, -0.881530225276947, -0.1680949330329895, -0.0425739623606205, 0.10345905274152756, 1.0657556056976318], [-1.5502113103866577, 1.4825472831726074, 0.10088315606117249, 0.6170859336853027, -0.6036269068717957, -0.05759408697485924, -1.8780019283294678, -2.2834365367889404, 0.20220431685447693, 0.37714987993240356, -1.9337769746780396, 0.40909603238105774, -1.0630359649658203, -0.020046940073370934, -1.0476897954940796, -1.5758445262908936, -0.0679578110575676, 0.523706316947937, -0.7865052223205566, -0.8275421857833862, -1.4698511362075806, -0.6766619682312012, -0.02129349112510681, 0.9493621587753296, -1.4939157962799072, 0.19913800060749054, -0.0996006578207016, 1.6444064378738403, 1.1821515560150146, 0.98410564661026, -0.2461499273777008, -1.5868027210235596, -1.5254418849945068, -0.017904935404658318, 1.4936537742614746, -1.1737209558486938, 0.9179073572158813, 1.1842774152755737, -1.1887562274932861, -2.2460432052612305, -1.0105676651000977, -0.3751642405986786, -0.010940145701169968, 0.2549564838409424, 0.10623946040868759, 0.39550501108169556, 0.6218819618225098, -0.056547410786151886, -0.4755379259586334, 0.08380588889122009, 0.17225497961044312, -1.2591816186904907, -1.533339262008667, -0.755538284778595, -1.4525138139724731, -0.9765708446502686, -0.4312755763530731, -2.173318862915039, -0.7493900656700134, 0.6259965896606445, 0.010228006169199944, 0.7334831953048706, 1.5637775659561157, 1.7775788307189941], [0.8380244374275208, -0.708563506603241, -0.0495501309633255, 0.34031158685684204, 0.48714324831962585, -2.6340675354003906, -0.349625825881958, 0.31613627076148987, -0.058854840695858, 0.5611923933029175, 0.093580462038517, 0.06813644617795944, -0.9737869501113892, 0.19784832000732422, -0.6600075364112854, -0.19285213947296143, -1.4971773624420166, -0.5412209630012512, 1.5943173170089722, 0.9477548599243164, 0.5288897156715393, -0.08138926327228546, 0.7619550824165344, -0.6092588901519775, 0.6212829947471619, 0.20411694049835205, 0.990865170955658, 0.34319114685058594, -1.2261203527450562, 0.07874651998281479, -0.726315975189209, 0.5482063293457031, 0.6759935021400452, -1.2695634365081787, 0.5334656238555908, -0.8193885087966919, 0.48259827494621277, 1.5943692922592163, -0.36317500472068787, 1.3973673582077026, 1.188230037689209, 1.637122631072998, -0.22720873355865479, 0.6010230183601379, -1.1423152685165405, 1.5501892566680908, 1.1519743204116821, -2.3497066497802734, -0.9960376620292664, 2.6395013332366943, 0.48331382870674133, 0.7377853989601135, -0.11867335438728333, 0.39973700046539307, 0.6992087364196777, 0.8456144332885742, 0.21975047886371613, -0.36381009221076965, 0.7475517988204956, 1.9168884754180908, -0.8118771910667419, -0.22779978811740875, 0.45371004939079285, -0.9882873892784119], [-0.6997069716453552, 0.8608332276344299, 0.1978444904088974, 0.8363684415817261, -0.10475941002368927, -0.6540502309799194, 1.086743712425232, 0.9271790385246277, -0.30494970083236694, -0.4396665096282959, -1.3700683116912842, -1.1639392375946045, -0.26565200090408325, 1.401700496673584, -0.36674317717552185, 0.7283892035484314, 0.1726277768611908, 1.8991562128067017, -1.6921086311340332, -0.34943386912345886, -0.376375675201416, 0.8453503847122192, 0.8308641314506531, 0.4718419313430786, 1.2126060724258423, 0.05488841235637665, -0.4751715362071991, -0.35613104701042175, 0.2196912169456482, -0.5139781832695007, -0.2693846821784973, 0.8601569533348083, 0.9877604246139526, 0.5110591650009155, 1.368534803390503, 0.5084982514381409, 0.45546647906303406, 0.29112690687179565, -0.48903852701187134, -0.5142372846603394, 0.9771090149879456, -0.005458136089146137, -0.0836726576089859, -0.11156594753265381, -1.3724523782730103, 1.1637479066848755, 1.0429928302764893, 0.6155108213424683, 0.4647410809993744, -0.22499683499336243, 0.09545022249221802, 1.2702499628067017, 0.6246610879898071, 0.8804497718811035, 0.6920263171195984, 0.5929694175720215, -0.5678296089172363, -0.7209002375602722, -0.5929935574531555, 1.7518218755722046, 0.8059931993484497, -1.292040467262268, -0.5958924293518066, 0.0010965055553242564], [-1.6640836000442505, -1.05667245388031, 0.7718484401702881, -0.586678683757782, -1.1862481832504272, -0.09123416244983673, 0.25034746527671814, -0.3978571891784668, -1.3741475343704224, -2.6796658039093018, 1.6451406478881836, 0.5130574703216553, -0.11063850671052933, -0.7963268160820007, 0.6852217316627502, -0.8048455119132996, 2.0350003242492676, 0.20891651511192322, 1.4456405639648438, -0.5339022278785706, -0.5519675016403198, 0.9956164956092834, 2.5359930992126465, -0.8078603148460388, 0.8044431209564209, -2.1674656867980957, 0.4697326123714447, 2.6121320724487305, 1.2898874282836914, -0.44306010007858276, 0.5449182987213135, 0.8889427185058594, -1.1168946027755737, -0.006814931053668261, -0.17101497948169708, -0.5691860318183899, 0.8454204201698303, -0.8784239292144775, -0.023526787757873535, 0.3680248558521271, -0.11125753074884415, -1.0455456972122192, 0.8218880295753479, 0.22724196314811707, 1.1828893423080444, 0.42491739988327026, -0.501166820526123, 1.2712393999099731, -1.6717865467071533, 0.23836985230445862, 0.08273134380578995, 1.7072980403900146, 1.8990203142166138, -0.9523314833641052, 1.7405794858932495, 0.5818631649017334, 0.49255821108818054, -1.1771738529205322, 0.00433841859921813, -0.039930444210767746, 0.054252102971076965, 0.31458497047424316, 3.721658706665039, 0.2897595167160034], [-0.41933318972587585, 0.7758185863494873, -2.534489393234253, -0.8473126292228699, -0.6793853640556335, 0.6676675081253052, -1.766733169555664, -1.2008392810821533, -0.5393686294555664, 0.048898596316576004, -0.8813122510910034, -0.5547704100608826, 2.0128610134124756, 0.38289451599121094, 1.2811555862426758, 0.9780211448669434, -0.11581675708293915, -0.7114430665969849, -0.2644636631011963, -1.5265806913375854, 0.8465223908424377, -0.12373030185699463, 0.6813868284225464, -0.11732695996761322, -0.6062421798706055, -0.821858286857605, -1.9232006072998047, 0.6959823369979858, -0.5622009634971619, -0.07227092981338501, -0.35183849930763245, 1.987975001335144, 0.4286208748817444, -0.6897563934326172, 0.48968827724456787, 1.2123724222183228, 0.8263673782348633, 0.3618870973587036, 0.052208393812179565, 1.3212097883224487, -2.223447799682617, 0.916312038898468, -1.7994078397750854, -0.05029444769024849, -1.0074001550674438, -0.16323600709438324, -0.8713660836219788, -0.9112226963043213, 0.7977333068847656, -0.5405649542808533, -0.28860560059547424, -0.47664570808410645, -1.0129830837249756, -0.8124752640724182, -0.6737394332885742, 0.5346278548240662, -0.8098656535148621, 0.0858527347445488, -0.9065214991569519, 0.20189838111400604, 0.37086424231529236, 0.9417996406555176, 0.3546409606933594, 1.6249171495437622], [1.4348840713500977, 1.248053789138794, -0.6505947113037109, -0.5773927569389343, -1.3945194482803345, 0.1339792162179947, -0.7252760529518127, 0.4802696108818054, 0.007759036961942911, -0.6097040176391602, -0.1836196780204773, 2.783252239227295, 0.10193082690238953, 0.0280340313911438, 0.1401471346616745, -0.27002593874931335, 1.659123182296753, 0.7346534729003906, 0.44177937507629395, 0.732811450958252, -0.47459742426872253, -0.7393245697021484, 0.35863247513771057, 0.1643964946269989, -0.5348486304283142, -0.06333974748849869, 0.8358328342437744, -1.0767450332641602, 0.6621280908584595, -0.5017900466918945, -0.5614084005355835, 0.9654355049133301, 0.7967157363891602, 0.30941012501716614, -0.4048578143119812, -0.7248456478118896, 0.17588376998901367, -0.43985655903816223, 0.21459844708442688, -0.2521125376224518, 1.0318711996078491, -1.0747992992401123, 0.6425040364265442, 0.16333742439746857, -0.770769476890564, 0.4093312621116638, -1.1757310628890991, -1.0964186191558838, -0.6600379347801208, -0.017981812357902527, 1.3272336721420288, 0.13862501084804535, -1.4658551216125488, 0.30733639001846313, -0.3703424632549286, -1.1144167184829712, 0.4712010324001312, -0.6329335570335388, -0.14316920936107635, 0.27426615357398987, -0.23640812933444977, -0.8888833522796631, 1.0051325559616089, -0.7190267443656921], [-0.8233851790428162, 0.05454924702644348, -0.288727343082428, -0.1599421352148056, 1.2220065593719482, 1.0040256977081299, -0.28000202775001526, -2.156367540359497, 2.1081926822662354, 0.2434149831533432, 0.6696528196334839, 0.8078886866569519, 0.5432182550430298, -0.19857263565063477, -1.2775133848190308, -1.9005942344665527, -0.08454468846321106, -0.8691089153289795, -0.5381391048431396, -0.14396080374717712, -0.17591586709022522, -0.015445894561707973, 0.590364396572113, 0.38230353593826294, -0.6251578330993652, -0.37721171975135803, 0.5396042466163635, -0.3900362551212311, 0.9988413453102112, 0.051021575927734375, 0.5080607533454895, -0.37890514731407166, -1.3706856966018677, 1.0782266855239868, -0.20351503789424896, -1.6148784160614014, 0.060651298612356186, 0.6066526174545288, 0.5378398299217224, -1.0250991582870483, -0.5789472460746765, -1.6906509399414062, 0.08162190020084381, -0.516196072101593, -1.3529454469680786, 1.6484713554382324, 0.09368973225355148, 0.2885547876358032, -0.01493899803608656, 0.4691040813922882, 0.35681214928627014, -0.0403798446059227, -0.22704869508743286, -0.842796266078949, -0.8013436198234558, -0.11549920588731766, 0.4703504741191864, 0.12093497067689896, -1.1281014680862427, 0.3607054352760315, -0.5940445065498352, 0.21448977291584015, 0.4784740209579468, 0.48116979002952576], [0.5904394388198853, 0.9767721891403198, -0.2502695322036743, 1.050796389579773, 1.242775321006775, 1.1027677059173584, 0.7117967009544373, 0.46491795778274536, -0.9379309415817261, 0.5330191850662231, 0.2177663892507553, -1.2933725118637085, -0.7709588408470154, 0.7998578548431396, -1.9641813039779663, -0.23714135587215424, 0.3140988051891327, -0.3500555157661438, 0.544318437576294, 0.3947601914405823, -0.7091739177703857, -1.36702561378479, 0.457653284072876, 1.0747456550598145, 1.5470998287200928, -1.9718787670135498, -1.6614267826080322, 0.2659378945827484, 0.6368803977966309, 1.0963304042816162, -0.08485139161348343, -0.433807373046875, 0.4732067883014679, 0.6042414903640747, 0.17784908413887024, 1.1307284832000732, 0.33705827593803406, -1.849161982536316, -0.5099878311157227, 1.2874094247817993, 0.4655621647834778, -0.2016715556383133, -0.877275288105011, 0.7411537170410156, -1.3919472694396973, 1.7871934175491333, -1.9440505504608154, 0.5898832082748413, -0.61198490858078, 0.09630957245826721, 1.1435104608535767, -0.5926379561424255, -0.3048263192176819, 0.26365926861763, -0.36870619654655457, -1.9237085580825806, 0.4760707914829254, -0.5506871938705444, -0.640098512172699, 1.0472824573516846, -0.48319071531295776, 0.001997480634599924, -0.2516805827617645, -1.3354331254959106], [0.0043922243639826775, -0.1389690339565277, -1.1808586120605469, 1.7271006107330322, 1.8969457149505615, -1.1631457805633545, -0.044200342148542404, -0.42997002601623535, 0.21538303792476654, 0.3927522599697113, -0.23814280331134796, 0.15377958118915558, -0.4360637068748474, -0.33485063910484314, 1.3875101804733276, -2.0473592281341553, -1.0136966705322266, 0.4011569321155548, 0.6348795294761658, 0.6175865530967712, 1.1935063600540161, 0.2263808250427246, -0.2863546907901764, 0.3719724118709564, -0.4744652211666107, -0.043190836906433105, -0.2728787660598755, -0.3897722661495209, 0.2393091320991516, -1.34807288646698, 0.3067680299282074, 0.637454628944397, -1.864349603652954, -1.1433436870574951, -0.19489343464374542, 0.27081531286239624, 2.193121910095215, 0.516652524471283, -0.6171795725822449, -0.7233027815818787, -0.8832105994224548, -0.8736645579338074, -0.034136638045310974, 1.0946648120880127, -1.4262185096740723, 0.868581235408783, 0.45452022552490234, 1.9616400003433228, 0.27517208456993103, -1.4191575050354004, 0.6739253401756287, 0.5665104985237122, -1.1631324291229248, -0.7982534766197205, 1.1178289651870728, 1.6370834112167358, 0.4559463858604431, 1.773140788078308, -0.8262382745742798, 0.10958430916070938, -0.6858944892883301, 0.23944701254367828, -1.3621413707733154, -2.8656461238861084], [0.40765923261642456, -1.1970456838607788, 0.17421551048755646, -1.486100435256958, -0.4504251182079315, 0.33149251341819763, -0.12612920999526978, 0.9501064419746399, 0.027602529153227806, 2.4067957401275635, 0.012572446838021278, 0.22740881145000458, 1.044456124305725, 0.7823919057846069, 0.7814239263534546, 0.83943110704422, -0.5477020740509033, 0.39660391211509705, -0.09075868874788284, 0.3608032763004303, 0.3948759436607361, -1.516984462738037, -0.808494508266449, 0.09532766789197922, -1.040854573249817, -0.5547136068344116, 0.6458523273468018, 0.22796064615249634, 0.24548356235027313, 1.6611770391464233, -0.09153851121664047, 2.325515031814575, -0.5742954015731812, -0.6158467531204224, 1.060516119003296, -0.33932751417160034, -0.5047276616096497, -0.5991835594177246, -0.757229745388031, -0.34080010652542114, 0.56715327501297, -0.09830357134342194, -0.7552618980407715, -0.3023432195186615, -0.11200502514839172, -0.4854675531387329, 0.6322527527809143, -2.016160488128662, 0.24948063492774963, 0.7572605013847351, 1.1131278276443481, -1.6624109745025635, -1.9433104991912842, 1.383405089378357, -0.41078323125839233, -0.2916393280029297, 0.8030105829238892, -0.33011335134506226, -0.012784204445779324, 0.6245244145393372, 0.6867825984954834, -0.2755640745162964, 0.942487895488739, 1.6762826442718506], [-1.914289951324463, 0.13422814011573792, -0.302587628364563, 1.9105901718139648, 0.6087713241577148, 1.381077527999878, -0.7516429424285889, 0.4504721760749817, 1.9368723630905151, 0.45974016189575195, 0.25599774718284607, 0.15781530737876892, 0.19552277028560638, -0.10809918493032455, -0.074446901679039, 0.9997749328613281, -2.179537057876587, 0.43367451429367065, 0.15344101190567017, 0.2680668532848358, -1.2412606477737427, -0.2627994120121002, 0.2623308598995209, -1.8035850524902344, 0.10417357087135315, 2.3539621829986572, -0.4597274363040924, 0.045431822538375854, -0.014386662282049656, -0.1018947958946228, -1.3480920791625977, 1.0373998880386353, -1.1346312761306763, -0.3862314224243164, -0.9513559937477112, 0.10523742437362671, 0.14243200421333313, 0.18602101504802704, 0.01737569272518158, -0.2297901213169098, 1.3509949445724487, 0.17644566297531128, 0.6768956780433655, -0.641998827457428, -1.2584820985794067, 0.5464991927146912, -0.9167982339859009, 0.6522142291069031, 0.2741073966026306, -0.012960638850927353, 1.2585755586624146, 1.5797030925750732, -0.14055976271629333, 0.3989303708076477, 0.382188618183136, -0.9119174480438232, 0.38196250796318054, -0.9619486927986145, 0.014525990933179855, -0.5584835410118103, -1.0527458190917969, -0.7041923403739929, -0.4269416034221649, 2.5319488048553467], [0.41932761669158936, -2.389331817626953, -0.48402583599090576, -0.3153637945652008, 1.8466215133666992, 0.12440570443868637, 0.12103167176246643, 0.44444775581359863, 0.3676178753376007, -0.3306277394294739, -0.4509359896183014, -1.2998708486557007, 1.5905143022537231, 0.5640134215354919, -0.09758609533309937, 0.10266253352165222, 0.21641941368579865, -1.5515047311782837, -0.16805584728717804, -1.6012016534805298, 0.9439489841461182, -1.2863737344741821, -0.8800087571144104, 0.8825852274894714, -0.19461767375469208, -1.7892117500305176, 0.5286513566970825, -0.78226637840271, -0.20178577303886414, 0.7187751531600952, 0.8840782642364502, 1.047713279724121, -0.8258350491523743, -0.039732884615659714, 0.8490986824035645, -0.16827590763568878, -0.9813753962516785, 0.5471170544624329, -2.1429603099823, 0.10773131996393204, 0.18899643421173096, -1.0303449630737305, 2.1595325469970703, 1.632428526878357, 1.490407943725586, 0.30333274602890015, -0.2472001165151596, 0.5681037902832031, -0.18946833908557892, 0.3372790217399597, 0.22280795872211456, 0.38106483221054077, -0.21006563305854797, -0.6960355639457703, 0.7858187556266785, 0.3817926049232483, -0.14432886242866516, 1.116959571838379, 1.4961081743240356, -0.7632109522819519, 1.5912023782730103, 1.0103399753570557, -1.7624469995498657, 0.4085117280483246], [-0.13642044365406036, 0.9234631657600403, -0.3920585811138153, 0.26667484641075134, -0.5941895246505737, 0.08546148985624313, -0.22886373102664948, 0.04771736264228821, -0.5282492637634277, 0.8482243418693542, 1.6263316869735718, -0.19876933097839355, -1.0534578561782837, 0.7464219331741333, 2.116961717605591, -1.813934326171875, 0.0768534243106842, -0.08182907104492188, -2.243197441101074, 0.7349174618721008, 1.4580155611038208, -0.35233935713768005, -0.5497073531150818, -1.1119894981384277, 1.3942642211914062, 1.2369203567504883, -0.43732717633247375, 1.767233967781067, 0.09399109333753586, 1.0135530233383179, 1.2480690479278564, 1.296839714050293, -1.439064621925354, 0.01630552113056183, 0.1635589599609375, -1.6245286464691162, 1.4834136962890625, 0.8895604014396667, -1.1369085311889648, -0.027157051488757133, 0.7028157114982605, 0.8275719881057739, 0.8093423247337341, -1.2730963230133057, -1.0189918279647827, -0.01075683906674385, -0.5693257451057434, -1.3756517171859741, -0.22385334968566895, -2.1974096298217773, -1.575326681137085, -1.3267357349395752, 0.6039128303527832, 2.397801637649536, -0.8416517376899719, 0.22949306666851044, 0.423616886138916, -1.2161391973495483, -0.28100645542144775, 0.8952953219413757, -0.6584935784339905, 0.2926355302333832, -0.910365104675293, -0.24472947418689728], [0.4648772180080414, 0.24657605588436127, -1.0670486688613892, -2.425429582595825, -0.41843852400779724, 1.7671054601669312, -1.7890757322311401, 0.8012523651123047, -0.8608378767967224, -0.16166579723358154, -0.8593013882637024, 1.3302556276321411, 0.3066934049129486, 0.03289900720119476, 0.46044760942459106, -0.6031216382980347, -0.20048947632312775, -1.007450819015503, -0.6970588564872742, 0.7313371300697327, 0.6290334463119507, -0.5052108764648438, -1.9335229396820068, -1.4020353555679321, 0.9116216897964478, -0.03642012178897858, 0.21840506792068481, 1.190294623374939, -0.6132274866104126, 0.8902841210365295, -0.7175883650779724, -0.5394433736801147, 0.3925374746322632, -0.22555695474147797, 1.1915887594223022, 0.14851243793964386, 0.4329725503921509, -1.1854490041732788, -1.5921701192855835, 0.08762490004301071, -0.5654230713844299, 0.20380502939224243, -0.5543502569198608, 0.4243142902851105, 0.07536177337169647, 0.09748473763465881, 0.2161020189523697, 0.16900134086608887, -0.41758519411087036, -0.9857452511787415, 2.3936421871185303, 0.06031603738665581, 0.06517583131790161, -0.018066002056002617, -0.5355440974235535, 0.03979690372943878, -2.0351078510284424, -0.182950958609581, 1.6589816808700562, 0.31996220350265503, 0.35310056805610657, -0.36999186873435974, 0.5374258756637573, -1.7169690132141113], [0.027226436883211136, -2.2665467262268066, -0.5748112201690674, -1.0391069650650024, -0.42541441321372986, -0.6016645431518555, 0.602653980255127, -0.3696601986885071, 0.11411131918430328, -0.7161645889282227, 0.07324058562517166, 0.4775787889957428, 0.7557962536811829, 1.253239393234253, -0.8093592524528503, -0.0034672401379793882, 0.2140117883682251, 1.2135517597198486, 1.1182571649551392, 0.6804275512695312, -0.3586413860321045, -1.7189099788665771, -0.5670384168624878, 0.606621503829956, -0.436657190322876, 1.8768197298049927, 0.20837853848934174, -0.4108866751194, 0.06990921497344971, 1.0156421661376953, -1.8106367588043213, 0.0764959380030632, -0.7638383507728577, 0.1727483868598938, 0.5312013626098633, 0.5718514323234558, -1.9301437139511108, -0.8556025624275208, -0.9453442096710205, -0.9504525065422058, -1.6298949718475342, 0.09527532756328583, -0.45536863803863525, 0.5342589616775513, 0.4802533686161041, -2.1328632831573486, -0.8823783993721008, -0.06847680360078812, -0.31499600410461426, -1.6893410682678223, 1.0972723960876465, -0.927318274974823, -0.9707210659980774, 0.5244598388671875, -0.02550814114511013, 0.6104297041893005, -0.7734607458114624, 0.5972050428390503, -0.5659897327423096, -0.3471306562423706, -0.2169104367494583, -0.5350676774978638, -2.1029396057128906, -0.05189693346619606], [-0.6948531270027161, 0.33291304111480713, 1.254526972770691, -1.2753019332885742, -2.067854642868042, 1.547762155532837, 2.679967164993286, -0.3257520794868469, 1.056206464767456, -2.064305067062378, 0.28547513484954834, 1.2712814807891846, 0.965230405330658, -0.5311320424079895, -0.79398512840271, -0.5520956516265869, -1.0658559799194336, -0.530158519744873, -2.0439534187316895, 0.9971347451210022, -0.609453022480011, -0.0048952349461615086, -0.03385545685887337, 0.7465803027153015, -0.2229749709367752, -0.880139172077179, 0.15768864750862122, -2.671257495880127, 0.44367024302482605, 1.2875674962997437, -0.28069257736206055, 1.5120104551315308, -0.6230513453483582, 1.3843553066253662, 0.18880411982536316, 1.12290620803833, -0.2484188973903656, -0.7308014035224915, 0.12579859793186188, -1.082483172416687, 1.0249245166778564, 0.7997771501541138, 0.41802966594696045, 1.615247368812561, -0.737208366394043, 0.9471021890640259, 0.244940847158432, 0.8543448448181152, 0.011183636263012886, -2.0709216594696045, -0.02494794689118862, 1.149601936340332, -0.8833540081977844, 1.7317192554473877, -0.9544575214385986, 0.6125344038009644, -0.12580740451812744, 1.1492899656295776, -1.4877212047576904, 1.133870244026184, -0.39490100741386414, -0.03798670321702957, 0.058127593249082565, 0.7303649187088013], [0.8536097407341003, 0.6489269137382507, 0.25339120626449585, -0.3172692656517029, -1.248024344444275, 0.04178832471370697, -0.812759280204773, 1.4066119194030762, -1.542636752128601, -1.0253448486328125, -1.2549656629562378, 0.4319329261779785, 0.953007161617279, -0.3906645178794861, -0.3246194124221802, -0.33236968517303467, -1.681653380393982, 1.8226757049560547, 0.3819667398929596, -0.13170848786830902, 1.405043601989746, -1.129814624786377, -1.3978911638259888, 1.9848957061767578, 0.399041086435318, -0.34259986877441406, -0.9924863576889038, -1.210376501083374, -0.08605638891458511, 1.578279972076416, 0.010519340634346008, -0.5534284710884094, -0.3548440933227539, -0.12673160433769226, 2.2333197593688965, -0.622260332107544, -0.2909356951713562, -0.15251493453979492, -1.8949116468429565, 0.1396097093820572, -0.7888262271881104, -0.31255006790161133, 1.5121508836746216, -1.3112746477127075, 0.2885419428348541, 1.9962636232376099, 1.5719051361083984, 0.7089613676071167, -0.38439813256263733, -0.6487436294555664, -1.2350997924804688, 0.6651908755302429, 0.3129357397556305, 1.1710197925567627, 0.8678598403930664, 0.44769027829170227, 0.30668574571609497, 0.6717773675918579, -2.5273237228393555, 2.1831061840057373, 0.19720885157585144, 0.08180427551269531, -0.03638578951358795, -0.6684836149215698], [-0.9096276760101318, -0.16716794669628143, -0.7071330547332764, 2.0322844982147217, 0.036006517708301544, -1.4907901287078857, -0.6594436764717102, -0.4614083766937256, 0.2302611917257309, -0.24656374752521515, 1.5518544912338257, -0.6245483756065369, 0.3765951693058014, 0.3315231502056122, 0.5725499987602234, 0.8672070503234863, 0.7823028564453125, 1.549377679824829, 0.7227987051010132, -1.037024974822998, -0.686528205871582, -0.8033738732337952, -0.4043262302875519, -0.9562883377075195, 0.6271052956581116, 1.466027855873108, -0.4802218973636627, -0.07718703150749207, 1.417624592781067, -0.30163076519966125, -0.34492892026901245, -0.9046023488044739, 2.4480299949645996, -0.002459745155647397, 1.2619832754135132, 1.3771235942840576, -1.0861470699310303, 0.09936243295669556, -1.5676755905151367, -0.03178444877266884, -1.4653217792510986, -0.45793190598487854, 0.7145171761512756, 0.09711600840091705, 0.35207316279411316, 0.348222553730011, 0.4382100999355316, 1.0473591089248657, -0.8948829770088196, -0.1827784925699234, 1.1990604400634766, 0.4339275658130646, 0.9300342202186584, 0.17484012246131897, 0.8690746426582336, 0.9002397060394287, 1.1621814966201782, -0.9537237286567688, 0.16157956421375275, -1.2489088773727417, -0.12619717419147491, 0.4465838372707367, -0.7285984754562378, -1.094606637954712], [0.5397546291351318, 0.8580307960510254, -0.565276026725769, 0.2889825999736786, 0.1904548853635788, 0.518734872341156, -0.22381983697414398, -0.7887545228004456, 0.0802115723490715, -0.02246747724711895, -0.05603525787591934, -0.5808674097061157, -2.1469249725341797, -1.175345540046692, -0.20728327333927155, 0.04285692051053047, -0.6296976804733276, -1.2088432312011719, 0.4312891960144043, 0.10134774446487427, 1.175786018371582, -0.6309762001037598, -0.1510782539844513, 0.4671134948730469, -0.8297926783561707, -0.9047489762306213, -0.6584189534187317, 0.13745427131652832, 0.9582369327545166, -0.9507250189781189, 0.6674700379371643, -1.1568946838378906, 0.06712068617343903, -1.9457342624664307, -0.6331548094749451, -1.0053527355194092, 2.764993190765381, -0.18194502592086792, 0.3378082513809204, 0.7067112922668457, 0.40597131848335266, -0.7000665068626404, -0.6115198135375977, 1.5179870128631592, -0.7308478355407715, -0.1561022251844406, -0.145316019654274, 0.33679211139678955, -1.2812490463256836, 0.08636187016963959, -0.038370151072740555, -0.6647457480430603, -0.9773774743080139, -0.9784648418426514, 1.4540077447891235, -1.5015681982040405, -0.5287603139877319, 0.760496199131012, -0.6030282974243164, 1.380820631980896, -0.2680634558200836, 0.852493166923523, -0.7449250221252441, 0.18630653619766235], [-0.9095499515533447, -0.9590778350830078, 1.1988416910171509, 0.8592231869697571, -1.7632520198822021, 0.21164865791797638, -0.24714395403862, 0.7829177379608154, -0.34915849566459656, -0.3480490744113922, 0.4258173406124115, 1.380947470664978, 1.7870432138442993, -2.1674914360046387, -0.6101158261299133, -1.8347454071044922, -0.34011930227279663, -0.2225404679775238, -0.5438650250434875, -0.8691189885139465, 1.310537576675415, -0.36897000670433044, 0.5598867535591125, 0.25445184111595154, 0.235051229596138, 0.4295355975627899, -0.22745674848556519, -0.4962955713272095, -1.7060779333114624, 2.026080369949341, 2.0475940704345703, 0.26175251603126526, -0.7103240489959717, 0.8780075311660767, 0.8070802688598633, 0.10655637085437775, -1.116944670677185, -0.6485357284545898, -1.4710170030593872, 0.11775336414575577, 1.3143649101257324, 0.5161948204040527, -0.6330411434173584, 0.2853495180606842, 0.4286571145057678, 0.41481688618659973, -0.36686497926712036, 0.41818270087242126, -0.4063207507133484, -0.6413850784301758, -0.3597847521305084, 2.291623830795288, 0.8703781366348267, -1.148190975189209, 2.1022820472717285, -1.7240592241287231, 1.7254453897476196, -0.4952283203601837, 1.492863655090332, 1.3679484128952026, 0.11881572008132935, 0.9087583422660828, -0.07099688798189163, 0.6583333015441895], [-0.9168710112571716, -0.8363776206970215, -2.215090036392212, 0.697970449924469, -0.05166760832071304, -0.1972539722919464, 0.020603056997060776, 0.15272857248783112, 0.3214941918849945, -1.0379416942596436, 0.084878109395504, -0.5205771923065186, -0.8793003559112549, -1.8298752307891846, -1.4642988443374634, -0.8854995965957642, 1.3888753652572632, 1.099290370941162, 0.31463104486465454, 0.6090394258499146, -0.6512114405632019, -0.07865723967552185, -1.279669165611267, -0.885045051574707, 0.041849181056022644, -0.47843724489212036, -0.17765559256076813, 1.8381125926971436, -0.12509359419345856, -1.0578370094299316, 0.2728566825389862, 0.6600936651229858, 0.4619726538658142, 0.49506038427352905, 0.6785234808921814, 0.15002818405628204, -0.3164518177509308, -1.298927903175354, 1.006128191947937, -0.4178910553455353, 1.167770266532898, -0.6664118766784668, 2.018259286880493, 0.7334737181663513, -0.1309119611978531, -0.39875033497810364, 1.3461507558822632, 0.2618345320224762, 0.1547221690416336, 0.767587423324585, -1.4601043462753296, 0.0034779165871441364, 0.9737312197685242, -1.8918582201004028, -0.7623671889305115, 1.3112232685089111, 0.36289939284324646, -1.116662621498108, 1.3034934997558594, -1.221359133720398, 0.07173777371644974, -0.8538891673088074, 1.4170140027999878, -0.1749291718006134], [1.052786111831665, 0.08335519582033157, 0.0420905165374279, 0.2201945185661316, 0.5394072532653809, 0.9501101970672607, 1.9912168979644775, 1.2339513301849365, -0.33153918385505676, 2.17585825920105, 0.20772287249565125, -0.48323866724967957, 0.6855327486991882, 0.011976943351328373, 0.927725613117218, -2.425921678543091, -0.624783456325531, 1.5530647039413452, 1.1767629384994507, 1.375722050666809, -0.8262056112289429, 0.9551159739494324, -0.071937695145607, 0.8278272747993469, 2.1183691024780273, 1.3798024654388428, 0.338928759098053, 0.16945838928222656, 0.08593076467514038, -1.0812321901321411, 0.06545589119195938, -1.5795632600784302, -0.09332688897848129, -1.044123649597168, 2.8896336555480957, -1.0793160200119019, -1.5455857515335083, 0.8142279982566833, -1.2503927946090698, -0.04619130864739418, -0.69358891248703, -1.6527032852172852, 0.018498307093977928, 0.5183700919151306, -0.8430611491203308, -0.5541903376579285, 0.2588646113872528, -0.781991720199585, 0.8546599745750427, -0.03888874500989914, 0.318275511264801, 1.6914151906967163, 0.4418809711933136, -0.8303078413009644, -0.9887297749519348, 1.7473747730255127, -0.6635169982910156, -0.25385984778404236, 1.0439060926437378, 1.1885747909545898, 0.22647593915462494, -0.3331027030944824, -0.17080241441726685, 1.269364595413208], [1.1973063945770264, 1.5321884155273438, 0.5135945677757263, -0.16564224660396576, 1.9512757062911987, 0.06226857379078865, -0.5411443710327148, 1.1693973541259766, -0.2933303713798523, -1.473311185836792, -0.6662183403968811, 1.6708598136901855, -0.44434064626693726, -1.8860722780227661, -0.4244415760040283, -0.144948810338974, -1.1010982990264893, -0.8411446213722229, -1.0741631984710693, -0.05163295939564705, -1.6031782627105713, 0.39657843112945557, -1.2861899137496948, 0.8788754940032959, -0.3121149241924286, 0.5587984323501587, -0.1410103440284729, 1.142311692237854, 0.5500469207763672, -0.08090166002511978, -0.04186731576919556, -0.4948265254497528, -1.1696528196334839, 0.97708660364151, -0.022100405767560005, 0.2921658158302307, -0.23792636394500732, -1.9282763004302979, 0.21185685694217682, 0.46207165718078613, -0.16867414116859436, -0.1615147888660431, 2.5030829906463623, -0.872342050075531, 0.057362210005521774, 0.3868381083011627, -1.5652403831481934, -1.5297256708145142, -1.4169938564300537, 0.36486247181892395, 0.7811633944511414, -0.15352624654769897, 0.5943108797073364, -2.1767992973327637, -1.6235089302062988, -0.27270057797431946, -0.5300081372261047, -0.005356077570468187, 0.2945506274700165, 1.4381153583526611, 0.23229357600212097, -1.5871243476867676, -2.207571268081665, 0.1537644863128662], [0.1132965013384819, -0.11213728785514832, -1.2861156463623047, -0.47424960136413574, -0.655147135257721, 1.012934923171997, 1.8088370561599731, -1.0235649347305298, 0.29205605387687683, 0.49461841583251953, 0.5775688290596008, -1.0084741115570068, 1.429274082183838, 2.4035794734954834, 1.1717312335968018, 0.6316569447517395, -1.235124945640564, -0.7887653112411499, 0.29308316111564636, -0.054227739572525024, -1.473573923110962, 0.7793487310409546, 0.8618665337562561, -0.32739847898483276, 0.26142337918281555, 0.5450051426887512, 1.0069689750671387, -1.3795208930969238, -1.8730565309524536, 0.5613450407981873, 0.907787561416626, -1.98712158203125, -0.8688488602638245, 0.12753473222255707, 0.8132561445236206, 2.2420589923858643, -0.7912181615829468, -0.4326765835285187, -1.5533645153045654, 0.7019974589347839, 0.2828095555305481, -1.1749428510665894, 0.38842901587486267, 1.2600727081298828, 0.30290883779525757, -1.7492693662643433, -0.29637253284454346, 0.4241600036621094, -0.45201945304870605, -0.3664880692958832, -0.10863298177719116, 0.9575985670089722, -0.07929688692092896, -0.6304510831832886, -0.1845150738954544, 0.3718235194683075, 1.4229223728179932, -1.8829882144927979, 0.02527276985347271, 0.7295864224433899, 0.37393516302108765, -1.8094197511672974, -0.5421324968338013, -0.2569320499897003], [1.983152985572815, 1.7462958097457886, 1.3908464908599854, 0.5211074352264404, -0.12581048905849457, -1.15634286403656, 0.915377140045166, 0.5837374329566956, -2.057551860809326, 0.5995585322380066, -0.9469552636146545, -1.2653164863586426, -0.01565711572766304, -0.08981665223836899, 1.7267851829528809, -1.0616663694381714, 1.324485421180725, 0.4264747202396393, -0.8064738512039185, -1.3830606937408447, 0.2288157045841217, 0.031546711921691895, -0.17878030240535736, 0.06776978820562363, -0.1945771872997284, 1.9817217588424683, -0.6130731105804443, 2.167294979095459, -0.6324588060379028, 1.419012188911438, -0.3880447745323181, -0.8169675469398499, -0.10791525989770889, -0.5563029646873474, 1.3241405487060547, -2.3897838592529297, 0.268335223197937, 0.6853410601615906, -1.2971389293670654, 0.3624977767467499, 0.19385647773742676, -1.2346667051315308, 1.392635464668274, -1.1467353105545044, -0.5121616125106812, -0.8456089496612549, 1.7264665365219116, -0.6535267233848572, 0.0941491350531578, -2.627997398376465, 0.12381380796432495, 0.21166133880615234, 0.16036786139011383, 0.7307807207107544, -1.3391571044921875, -0.4311818480491638, 2.984755039215088, -1.9484297037124634, 0.5308152437210083, 0.113697350025177, -0.45428627729415894, 1.3666291236877441, 0.17447786033153534, -0.906006932258606], [0.5319219827651978, -1.2795699834823608, 0.06550264358520508, 0.2601740062236786, -1.7754946947097778, -1.3171073198318481, -1.311476469039917, -0.48681581020355225, 0.39754563570022583, -0.04812180995941162, -0.1124204471707344, -1.0275040864944458, 0.2250923216342926, 0.4196804165840149, 0.5858986973762512, -0.4319272041320801, -0.502834141254425, 1.1291316747665405, 1.9779471158981323, -0.5770008563995361, 0.37907490134239197, -0.20338350534439087, -0.8754693865776062, -1.543919563293457, 0.2581973671913147, -0.14886553585529327, 0.10414273291826248, 0.4021919369697571, 0.37152883410453796, 0.5656588673591614, -1.5847070217132568, -1.4019619226455688, 0.7116789817810059, 0.38243234157562256, -0.2491171509027481, -0.8099967837333679, -0.04097392037510872, 0.9337632656097412, 0.09230317920446396, 2.4546091556549072, 1.252358317375183, 0.2587644159793854, 0.29992252588272095, 0.39823228120803833, -0.6678227782249451, -1.2234162092208862, 0.40776240825653076, -1.4289023876190186, -0.09363474696874619, 2.1857380867004395, 1.6023542881011963, -1.0688730478286743, -0.5341759324073792, 0.2633659541606903, -0.5143776535987854, 0.5282568335533142, 0.820212721824646, 1.095589518547058, -0.36746829748153687, -0.808442234992981, -0.5344341993331909, 0.22585441172122955, 2.6798017024993896, 0.26052477955818176], [-0.14009258151054382, 1.3795030117034912, -0.25854411721229553, -0.45834222435951233, -0.9949474930763245, -1.0782020092010498, -0.8248830437660217, -0.5589118003845215, -0.029699573293328285, 1.7299052476882935, 0.07153517007827759, 1.2540338039398193, 0.052106741815805435, 0.8322051167488098, 1.0204817056655884, -0.3599245846271515, 0.20630936324596405, -0.31707289814949036, -3.5276970863342285, 0.6494163870811462, 0.8571928143501282, 1.6633318662643433, 0.48154863715171814, -0.011924178339540958, -0.8173647522926331, -0.31825166940689087, -0.19401492178440094, -0.4582648277282715, -0.36449912190437317, 1.2301976680755615, 0.1660478413105011, -0.17963822185993195, 0.3259766697883606, 0.5425019860267639, 1.5238043069839478, 2.2404725551605225, -1.8907526731491089, 0.020705100148916245, -1.6162766218185425, -0.7731263637542725, -0.44087085127830505, -0.8746494054794312, -0.554069995880127, 0.44826751947402954, 1.434667706489563, -0.20457084476947784, 0.3265329599380493, -1.3646692037582397, -0.22124022245407104, 0.532016396522522, 1.044847846031189, -0.43665942549705505, 2.0460281372070312, -0.6090905070304871, -0.8012932538986206, -0.10097646713256836, -1.6961283683776855, -1.3102927207946777, 0.22336359322071075, -0.7527505159378052, 2.2698094844818115, 1.4992921352386475, -0.23004767298698425, -0.09551145881414413], [-0.1106070950627327, -0.32369929552078247, 0.8732568025588989, 1.5242290496826172, -0.022525055333971977, 0.718131422996521, -0.11329512298107147, 0.0475291945040226, 0.6813675761222839, 1.0414471626281738, -0.21581773459911346, 0.9504791498184204, -1.4571956396102905, -0.09161379188299179, -0.12134788930416107, 0.13234756886959076, 0.5686514973640442, 0.45292261242866516, 1.2149628400802612, 0.5682547092437744, 1.4975292682647705, -1.5469391345977783, -2.1362733840942383, 0.1538449227809906, 1.0610814094543457, 0.7763752341270447, 0.24840395152568817, 0.8751842379570007, -0.35686612129211426, -0.41218483448028564, -0.46883878111839294, -0.20784761011600494, -0.4964507818222046, -0.2320578545331955, 1.014641523361206, -0.9017728567123413, -0.042151857167482376, 0.6125417947769165, -0.4168066680431366, 2.241334915161133, -0.2199775129556656, -0.6980419754981995, -0.2966121733188629, -0.27999281883239746, 1.1505078077316284, 0.5526157021522522, 0.5431987047195435, -0.620752215385437, -0.6217002868652344, 0.3972395658493042, -0.2464158684015274, -0.8222350478172302, -0.5546925663948059, -0.6631708741188049, 0.48967239260673523, 0.18938705325126648, 0.26384568214416504, 2.2070255279541016, 0.9079474210739136, -1.5242185592651367, 0.5688947439193726, 0.02055569924414158, 1.0934081077575684, 1.0127407312393188], [0.5025858879089355, 0.48103681206703186, -0.01363905519247055, -0.3070305287837982, 0.9617229104042053, 0.20065370202064514, -0.036150164902210236, 1.4219303131103516, -0.8286977410316467, 1.4548512697219849, 1.6289761066436768, 1.5008190870285034, -0.4267812967300415, 0.8663926720619202, -0.41828683018684387, 1.0483362674713135, -0.47349897027015686, -0.9676091074943542, -0.23303718864917755, 2.7047574520111084, 0.49820002913475037, -0.3172867000102997, 1.0880752801895142, 2.1609277725219727, 1.4026395082473755, 2.7363617420196533, -1.4997612237930298, -1.2481929063796997, 1.607846975326538, 0.9672377705574036, -0.38012635707855225, -0.9337485432624817, 2.703958034515381, 1.0437383651733398, 0.0009625226957723498, -0.2530486285686493, 0.13647893071174622, -0.9961910843849182, 0.25218114256858826, -0.08145594596862793, 0.34984877705574036, 1.0990960597991943, 1.0753360986709595, 0.7126210331916809, 0.9452964663505554, 0.19095833599567413, 0.7367255687713623, -0.7019244432449341, -1.2975084781646729, -0.24556079506874084, -0.5980633497238159, 1.6621580123901367, -0.19960461556911469, 0.8855873346328735, -1.6880884170532227, -0.8818763494491577, 0.328582227230072, -0.07395357638597488, -0.01639770343899727, -0.02208731137216091, -1.5832173824310303, -0.7937629222869873, -0.7837761044502258, 0.20949149131774902], [0.5186877250671387, -0.7450324892997742, -0.4916819930076599, 0.1482454538345337, 0.30534225702285767, -2.267768144607544, -0.6035414338111877, -0.1302562952041626, 0.9359707832336426, 0.457569420337677, 1.9922738075256348, 0.9609990119934082, 0.9030513763427734, 1.1185499429702759, 0.19467222690582275, -0.14266249537467957, 0.0890330970287323, 0.43416112661361694, -1.0906139612197876, 1.8272171020507812, 0.9715121984481812, -0.4954841136932373, -1.3400973081588745, 0.8592312335968018, -1.2319854497909546, 0.1825563758611679, 0.3152497410774231, 0.64569091796875, 0.4471991956233978, 1.6891282796859741, 0.4967513680458069, -0.1999768763780594, 0.720736026763916, -0.9207527041435242, -0.09280586987733841, -0.9389519095420837, 1.5080991983413696, 0.3592071235179901, -0.505980908870697, 0.5691354274749756, -0.677341878414154, 1.5528684854507446, -0.9446233510971069, -2.9577581882476807, 1.372954249382019, -0.5794863104820251, 1.4085108041763306, -0.045533694326877594, -0.3024168908596039, -0.25932446122169495, 0.17247867584228516, 0.6338564157485962, 0.2954857647418976, -0.8285127282142639, 1.9863625764846802, -0.17916588485240936, 0.21457505226135254, 0.03434532880783081, 0.1307888925075531, 0.12664178013801575, -0.061243169009685516, -1.6359390020370483, -0.5144770741462708, -0.4579034149646759], [0.3392103910446167, 0.14290383458137512, 1.3357409238815308, -0.7263802886009216, -0.44087234139442444, -1.21845281124115, 0.6647534966468811, 0.04423753172159195, -0.7610487341880798, -2.8808395862579346, 1.2200716733932495, -1.053186058998108, -0.8278641104698181, 1.8358408212661743, -0.8997516632080078, 0.7851346731185913, 0.3781014084815979, -0.08927498012781143, -0.06698267161846161, 1.0053232908248901, -1.4250283241271973, -0.4750641882419586, -1.2948931455612183, -0.6419187784194946, -0.507774829864502, 1.7685215473175049, 1.447872281074524, -0.803747832775116, -1.247558832168579, -0.4623311460018158, 0.15515784919261932, 0.3000386953353882, -1.1586782932281494, -2.9875071048736572, 0.03981458395719528, -1.247594952583313, -0.5770522952079773, 0.9509437680244446, -0.9290052056312561, 1.7080717086791992, -0.44803592562675476, -1.1885602474212646, -1.0071673393249512, 0.09388528019189835, 1.3116116523742676, -2.517279624938965, -0.785831868648529, 0.30671900510787964, 1.6525261402130127, 0.34869393706321716, 0.21511700749397278, -0.28415289521217346, 0.14591242372989655, -0.4779377579689026, -0.6425299048423767, -0.4824618399143219, -0.7674570083618164, 0.6289482712745667, 0.006015949882566929, 0.5826352834701538, 0.25092679262161255, -0.15861134231090546, 0.005347703117877245, 1.6577539443969727], [0.14309100806713104, 2.1474525928497314, 0.5570014715194702, 0.6187219619750977, 0.5672862529754639, 0.7833634614944458, 0.6396118998527527, -0.1969504952430725, -1.2046853303909302, 0.35558435320854187, 0.9789274334907532, -0.9111210703849792, 0.30180251598358154, 0.7502382397651672, 1.8789491653442383, -0.014107504859566689, -0.5170029997825623, -0.8646057844161987, 0.7678934931755066, 0.03716856613755226, 0.21575134992599487, 0.9890341758728027, -0.5740875005722046, 0.5041057467460632, 0.7642343044281006, -0.21042095124721527, -0.482228547334671, -0.12143220007419586, -1.6010624170303345, 1.1626865863800049, 1.0254932641983032, 0.1532658338546753, 0.20284807682037354, 1.7030620574951172, 1.3122111558914185, 0.7960437536239624, 0.5617400407791138, -0.38297560811042786, -0.21426568925380707, -1.0746861696243286, -0.12712416052818298, -1.0863027572631836, 1.5615392923355103, 0.07265588641166687, -0.4372827112674713, -0.4925791025161743, -0.6686913371086121, 0.27159780263900757, 0.7115002870559692, 0.012935810722410679, -1.1986653804779053, -1.083837866783142, 0.0030846805311739445, 0.5452175140380859, 0.4730018079280853, -0.6624515652656555, 0.0470263846218586, 0.28637516498565674, 0.0665392279624939, 0.2501918375492096, 1.0997650623321533, -0.10378526151180267, 0.37555909156799316, 0.3520911931991577], [-0.6118477582931519, 0.027215542271733284, 1.8591221570968628, -2.2878339290618896, -0.1567077934741974, 0.8897884488105774, -0.6602026224136353, 0.30062738060951233, -0.05957046151161194, 0.6645558476448059, -0.5555168986320496, -1.1397159099578857, 0.3567635118961334, -1.472219467163086, 0.25212574005126953, 0.3836798071861267, -0.058167699724435806, -0.09063281118869781, 1.632470726966858, 0.9612955451011658, 0.10767585039138794, -1.1732885837554932, 0.9101681113243103, 1.2535048723220825, -0.2908349633216858, 0.36109283566474915, -0.9041442275047302, -0.9661694765090942, -0.07885929942131042, -1.5378354787826538, -0.7039514183998108, -1.5250319242477417, -0.4560372829437256, 2.043923854827881, 0.7591584920883179, -0.9464636445045471, 0.8372922539710999, 0.6655253171920776, 0.967765748500824, -1.2254035472869873, -0.8092980980873108, -1.0239492654800415, -0.5026893615722656, 0.6800840497016907, -1.5623599290847778, 0.25176265835762024, -0.07271449267864227, -0.17715488374233246, -0.5612432360649109, 1.1880837678909302, 0.6236591935157776, 0.38990283012390137, -2.0748066902160645, 1.3014591932296753, 0.7867969870567322, -0.037671249359846115, 0.16884857416152954, -1.9207943677902222, 0.9831247925758362, -0.25912606716156006, -0.6430155038833618, -2.6520416736602783, -0.01448324415832758, 0.3170129656791687], [-1.2900142669677734, 1.3607114553451538, -1.546095371246338, 0.563309371471405, -0.4132818579673767, -1.0025193691253662, -0.29363900423049927, -0.6211448907852173, -0.7635016441345215, 0.9022188186645508, 1.672282338142395, 0.0839729756116867, -0.08624102920293808, -0.33672621846199036, -2.4042603969573975, 1.633872151374817, 0.2797754406929016, -1.6271016597747803, -1.7789093255996704, -1.0759291648864746, 0.9970090389251709, -0.4428271949291229, -2.002256155014038, -1.069258689880371, -0.6502541899681091, 1.4860442876815796, 0.3621830642223358, -2.6278443336486816, -0.8113657236099243, -0.34299445152282715, 2.77541446685791, 0.6595794558525085, 0.2421828955411911, 0.4741099178791046, 0.15796807408332825, 1.2428749799728394, 0.9697751998901367, 0.5652499198913574, 0.055816419422626495, 0.27978208661079407, 0.9556469917297363, 1.0016562938690186, 1.418480634689331, 1.8812994956970215, 0.11711902916431427, -0.3737601935863495, -0.09178723394870758, 1.2114067077636719, -0.17384767532348633, 0.633145809173584, -1.9647475481033325, 0.10836949199438095, 1.4748649597167969, 0.5267430543899536, 0.5323258638381958, 0.06824610382318497, 0.3284464478492737, -0.7678666710853577, 1.2587381601333618, -0.21835415065288544, -0.3034295439720154, 0.4840954840183258, 1.3151379823684692, -0.9180892109870911], [0.5312015414237976, 0.6064819693565369, 0.3910174071788788, -0.9535473585128784, 0.6667255163192749, 2.1718218326568604, 0.6341078877449036, -0.03558563068509102, -0.11127679795026779, -1.3851946592330933, -0.6371837854385376, -0.26736313104629517, 0.3821549415588379, -0.06214755401015282, -0.5315180420875549, -0.45904839038848877, -0.5621773600578308, 0.7681424021720886, 0.29042041301727295, -0.9542610049247742, -0.28270676732063293, -0.9217923283576965, -0.7383080720901489, 1.2372040748596191, -0.3742184042930603, -0.4919152855873108, -0.8748214244842529, -2.8086822032928467, 0.020521612837910652, 1.5541064739227295, -0.672953724861145, 0.5234259963035583, -0.25088343024253845, 1.209717869758606, -1.2280977964401245, 0.4225352108478546, 0.5396886467933655, -1.2348036766052246, -0.5819447636604309, -0.04574234411120415, -0.2228318601846695, 1.963513970375061, -0.06225716695189476, 0.9530316591262817, -1.6390873193740845, -0.7567272186279297, 0.4106428921222687, -0.16340035200119019, 0.9225990176200867, -2.1482961177825928, 2.007154941558838, 1.6241375207901, 0.30890631675720215, -0.00896354578435421, -0.6302062273025513, 0.5203996896743774, 0.6587164402008057, 1.1755805015563965, -0.4831101894378662, 1.0769362449645996, -1.771298885345459, -0.26959937810897827, -0.02814125083386898, 0.4336550831794739], [0.9407203197479248, -0.09022592008113861, 0.7901856899261475, 0.15546780824661255, 1.1016532182693481, 0.7098739147186279, -0.7471261024475098, 1.5737098455429077, -0.3817327916622162, 0.37445488572120667, -0.49117955565452576, 0.8802601099014282, -0.4275869131088257, -1.0713175535202026, -1.1897207498550415, 1.3098523616790771, -0.3741963505744934, 0.23079520463943481, 1.761710286140442, -0.07154456526041031, 1.6032006740570068, 0.20497702062129974, 0.3757829964160919, 1.7826199531555176, 0.5097414255142212, 0.005379720125347376, 0.1676464080810547, 0.0495416559278965, -1.2146164178848267, 1.189511775970459, 0.05105696618556976, -0.4009702503681183, -0.31487488746643066, 0.4517795443534851, -0.7931901216506958, -1.3505103588104248, -0.9293811917304993, -0.17019426822662354, -0.6210197806358337, 0.9906587600708008, 0.2703935503959656, -0.5292889475822449, -1.403484582901001, 0.3769822120666504, -0.7211487293243408, 0.9590234160423279, -2.0374748706817627, 1.6131811141967773, 0.2003050446510315, -0.7691832780838013, -1.3655532598495483, -0.4361940622329712, 1.42865788936615, -1.215111255645752, 0.07637810707092285, -0.5203555822372437, 0.3807416558265686, -0.01788976602256298, -0.4875255525112152, 0.393683522939682, -1.2137798070907593, -1.652523398399353, 0.3531719744205475, 0.5131102204322815], [1.4861520528793335, -0.4870206117630005, 0.06656944751739502, -1.2442065477371216, 0.2862703800201416, 0.16393694281578064, 0.43647634983062744, -0.09593908488750458, 0.6472027897834778, -0.36550620198249817, -0.8771239519119263, -0.1393306404352188, -0.9573287963867188, 0.8844597935676575, 0.7248251438140869, -0.6289941072463989, 1.8423304557800293, -1.2191781997680664, -1.1572961807250977, -0.8137688040733337, 1.2099590301513672, 0.5428057312965393, 0.04482397809624672, 0.33974945545196533, -0.19052572548389435, -1.3334617614746094, -0.8014218807220459, -0.34134718775749207, -1.9592465162277222, -0.09253043681383133, 2.051631450653076, -0.5522347688674927, -1.4793952703475952, 1.2735497951507568, 0.6989027857780457, -0.19311311841011047, -0.73179030418396, -0.1301121860742569, -0.20313149690628052, 1.3173426389694214, -0.7036423683166504, 1.4032341241836548, -0.0012928254436701536, 1.0071961879730225, 1.8044111728668213, -0.09894651919603348, -0.36952343583106995, -0.5413939952850342, -0.23803114891052246, -0.45739030838012695, -0.7088250517845154, 0.1581149697303772, 1.8413856029510498, 0.08973477780818939, 1.740278959274292, 0.08165328204631805, 0.20952686667442322, -0.1702871322631836, -1.7554320096969604, 2.072230577468872, 2.0701677799224854, -1.4582633972167969, 1.8337481021881104, -0.7020724415779114], [0.5462194681167603, -0.12471966445446014, -0.6174411773681641, -2.379530906677246, 0.6734517216682434, 1.1021411418914795, -0.8087654709815979, 0.12487071007490158, 1.4697281122207642, -0.30252805352211, -0.6800575256347656, -0.3476804494857788, -0.6947956681251526, 0.0796571671962738, -2.806318998336792, -1.8546963930130005, 1.0137107372283936, 0.8799700140953064, -0.02193467691540718, -0.05219784379005432, 1.9756473302841187, 0.3530888259410858, -2.3033783435821533, -0.7192267179489136, 1.8550195693969727, -1.3232320547103882, -0.6189725399017334, 0.8440634608268738, -0.2635732889175415, 0.7079716920852661, -1.014326572418213, 0.7306803464889526, -1.1612491607666016, 0.5484914183616638, -1.334275484085083, -1.102494478225708, 1.391739845275879, 1.1980178356170654, 0.31882187724113464, -0.5268777012825012, 1.172858715057373, -0.049515534192323685, -1.6265678405761719, -0.7509870529174805, 0.2600153088569641, 0.02767876349389553, 1.138051986694336, -1.7390046119689941, 0.9060011506080627, 0.2778615355491638, 0.4335097372531891, -0.15325330197811127, 0.7989069223403931, -0.905103862285614, 0.5743626952171326, -0.7687625885009766, -1.2999191284179688, 1.785983920097351, 2.352823495864868, -0.5756109952926636, 0.14309588074684143, 0.09417739510536194, 0.5835689306259155, -1.3255805969238281], [0.6013627648353577, 0.7065014243125916, 0.033447686582803726, -0.8693456649780273, 0.9204025268554688, 0.9635702967643738, -2.560913324356079, 1.4985991716384888, -1.4257433414459229, -0.006096103228628635, 1.0659067630767822, -0.8507282137870789, 2.309227228164673, -0.18141524493694305, -0.15792343020439148, 0.3059113919734955, -0.8261471390724182, 0.4712572991847992, -0.43377017974853516, 1.7826662063598633, 0.47571828961372375, 0.43052467703819275, 1.650134563446045, -1.3770076036453247, -2.1769163608551025, -0.5142736434936523, 0.3208465278148651, -0.12795482575893402, -0.001352909835986793, -0.60149747133255, -0.031651660799980164, 0.263795405626297, 0.045817989856004715, 2.2068169116973877, 0.4899180233478546, 0.9689196348190308, -0.7825449109077454, 0.34486693143844604, 1.0892226696014404, -1.1324267387390137, -0.17308716475963593, -0.3363470137119293, -1.672114610671997, -0.8079007863998413, 0.634836733341217, -0.17889389395713806, -2.0648210048675537, -2.1626622676849365, -0.4431098997592926, -0.33954840898513794, 0.4325100779533386, 1.6911511421203613, 0.6913920044898987, -0.055897071957588196, -0.4330603778362274, 1.2158880233764648, 1.4177305698394775, -0.6381812691688538, 0.1742357462644577, 0.49886050820350647, 0.7749085426330566, 0.49949178099632263, 0.6688923239707947, -1.0274397134780884], [1.9276721477508545, 0.29578346014022827, -0.7636923789978027, 0.3711841404438019, -0.6431074738502502, 2.6263506412506104, -1.443626046180725, 0.32355570793151855, 0.8927071690559387, 0.25732970237731934, 0.39963293075561523, -0.6643227338790894, 1.3145084381103516, -0.5121207237243652, -1.4316262006759644, -0.5316460728645325, -0.35640281438827515, 0.27174460887908936, 2.021973133087158, 0.8395261168479919, 0.020745495334267616, 0.6505072116851807, -1.1461836099624634, -0.13752281665802002, 1.3462715148925781, -0.4092034697532654, -0.3534773886203766, 1.3253990411758423, -0.4888680875301361, -0.1584872603416443, -1.205998420715332, -1.5525012016296387, -0.6965872645378113, -0.18595561385154724, 1.192076563835144, 1.2843068838119507, 0.023308271542191505, 1.383960247039795, -0.327304869890213, -1.7532119750976562, 0.005535305943340063, -0.9406150579452515, -1.1946579217910767, 1.3862638473510742, -0.11536893993616104, 1.335453748703003, -1.6398797035217285, -0.5464524626731873, 1.5064784288406372, 0.27520400285720825, 1.5655001401901245, 1.2637324333190918, 0.034489601850509644, 0.3026874363422394, -1.0821174383163452, 1.3243672847747803, 1.500395655632019, 0.10725963115692139, -0.2300398349761963, -0.21917979419231415, -1.37681245803833, 0.37797215580940247, -1.0861046314239502, 1.4957427978515625], [0.10008542984724045, 0.7821738719940186, -1.9287927150726318, 1.287796974182129, -1.2711114883422852, -0.44948112964630127, -0.6611343622207642, 0.06259247660636902, 0.024344012141227722, -2.6375322341918945, -0.007815062068402767, 1.3329129219055176, -1.7223930358886719, -1.338454008102417, -1.1042052507400513, 0.5379679799079895, 1.185072660446167, -1.1545264720916748, -1.4961156845092773, -1.325719952583313, -1.1752420663833618, -0.2838051915168762, 0.5441082715988159, 0.5545798540115356, 0.7743384838104248, 1.5930280685424805, -0.5078521966934204, 1.466095209121704, 0.06331781297922134, 1.2620432376861572, -1.003463864326477, -0.01420872937887907, 0.7931480407714844, 3.0277183055877686, 0.04196687415242195, 1.3676873445510864, 1.969708800315857, 0.03339596092700958, 1.283549189567566, 1.0503129959106445, -0.05801742523908615, 0.12710325419902802, 1.446652889251709, 1.55930757522583, 0.5272167921066284, -0.716686487197876, -0.6208345293998718, -0.3123423159122467, -0.6718738675117493, -1.0813486576080322, 0.5466436743736267, -1.2892552614212036, -0.7738152146339417, -0.29405102133750916, -0.7096654176712036, -0.03966901823878288, 1.176635503768921, -0.7746705412864685, -1.6435056924819946, 0.5588077902793884, -0.9389194846153259, 0.6608239412307739, -0.71852046251297, 0.303642600774765], [-1.9051793813705444, 1.6911273002624512, -0.515470027923584, 1.0237843990325928, 0.4620632529258728, -1.1207689046859741, -0.151658833026886, 1.1290298700332642, 0.2930155396461487, 0.7797365188598633, -1.5071487426757812, 1.8295212984085083, -0.41418886184692383, 0.6944840550422668, 1.5061057806015015, -0.9284285306930542, -1.2907500267028809, -0.3283509910106659, 0.055431053042411804, 0.39260050654411316, 1.2379943132400513, -1.59635329246521, 1.04890775680542, 0.3447687327861786, -0.045717839151620865, -0.14900656044483185, -1.0175637006759644, 0.09964348375797272, 0.6515145301818848, 0.8148094415664673, -1.3351504802703857, 0.6864016056060791, 0.8549448847770691, 0.02031065709888935, -0.26313289999961853, -0.4637463092803955, 0.4219292998313904, -0.48961615562438965, 0.6349166035652161, 0.515453040599823, 1.2142008543014526, -0.7549881935119629, -0.37995168566703796, 1.4303256273269653, -0.5732101202011108, -1.1477441787719727, -0.30490878224372864, -1.3743106126785278, 0.7200689911842346, 0.38167324662208557, 0.5384545922279358, -0.7973119020462036, 0.9431777596473694, 0.4294419586658478, -0.6146425008773804, 0.10392382740974426, -1.3597737550735474, -0.7988705039024353, -0.13904544711112976, -0.951974630355835, 0.3068746328353882, 0.5833576917648315, -0.6406046152114868, -0.4591304659843445], [1.104304552078247, -0.7227856516838074, -0.8808918595314026, -0.6195427775382996, 0.9825444221496582, -2.4332451820373535, -0.13647693395614624, -1.0547871589660645, -0.6089082360267639, -1.3659344911575317, 0.47532790899276733, -0.795921802520752, -0.6901035308837891, -0.18116338551044464, 0.5864671468734741, -0.14291511476039886, 0.12488643825054169, 0.18583109974861145, 1.0691190958023071, -2.024779796600342, 0.14781348407268524, -0.18575088679790497, 1.5742027759552002, 0.06935898959636688, -0.5864157676696777, 1.1565803289413452, 1.0009900331497192, 0.6524342894554138, -0.3440202474594116, 0.9005146622657776, -2.0403199195861816, 0.6495904922485352, -1.0050956010818481, 1.3059576749801636, -0.8051161170005798, 0.679948627948761, 0.27088648080825806, 0.24744626879692078, -0.9785127639770508, -0.2554134130477905, -0.4993168115615845, -0.3527717888355255, -2.054966688156128, -0.5281721949577332, -1.0363095998764038, -0.1870894581079483, 1.212003469467163, -1.586778998374939, 0.4838765859603882, -1.4396361112594604, 0.37701958417892456, 1.8110054731369019, 2.270042657852173, 0.04915536195039749, -0.5132360458374023, 0.2982856333255768, -1.343916893005371, -0.3101832866668701, 0.8024584054946899, -0.6768737435340881, 0.07597088813781738, 1.1360939741134644, 1.2440472841262817, -0.2818440794944763], [-1.0678505897521973, -2.8778293132781982, 0.4696170389652252, 2.735640525817871, 0.32820290327072144, -0.5428134202957153, 0.2684066891670227, 0.1420615166425705, 0.9513863921165466, 0.5849059224128723, -0.7922214865684509, 1.4267696142196655, -0.9825476408004761, -0.23502905666828156, 0.4873161017894745, -1.0644354820251465, -0.014459525234997272, -0.795974612236023, 0.40510937571525574, 1.1621097326278687, 0.3757263720035553, 1.9146376848220825, -0.21245044469833374, -1.6674808263778687, -0.8506779670715332, -0.7974383234977722, 0.7659109234809875, 1.167860984802246, -0.8205018043518066, 1.8424614667892456, -0.2131684124469757, 0.373292475938797, -0.671001136302948, 0.20419561862945557, -0.03758497163653374, 1.827345609664917, 0.4005677402019501, 0.32454735040664673, 1.3684072494506836, -1.2457016706466675, 0.2076680064201355, -0.9492577314376831, -0.27265653014183044, -0.7053332924842834, -0.9728997945785522, -0.9858810305595398, 0.49454084038734436, -0.937673032283783, 0.25367796421051025, -1.3430261611938477, -1.228850245475769, 0.08374025672674179, -1.230004906654358, -0.9500455260276794, 0.5951296091079712, -0.07297444343566895, -0.08745284378528595, -0.3775083124637604, 0.08900173753499985, -0.6253470182418823, -1.1559406518936157, 1.361099123954773, -0.1747300773859024, -0.3260258734226227], [1.412732720375061, 0.1222737580537796, -0.07014010846614838, 0.4181659519672394, -1.247697114944458, 0.7725757360458374, -0.6611793637275696, -1.551166296005249, -0.13527221977710724, -0.8772793412208557, -0.182006374001503, 0.5985206365585327, -1.581933856010437, 0.060739461332559586, -2.1918721199035645, 1.7035812139511108, 1.9384857416152954, -0.8228654861450195, 2.3440961837768555, -1.2945480346679688, 0.3031458258628845, -1.25630521774292, 2.749955415725708, 1.1626912355422974, 1.764973759651184, -1.1114470958709717, -0.7109501361846924, 0.029131513088941574, -0.2300235629081726, -1.0931967496871948, 1.5227311849594116, -0.5095456838607788, -0.1241406723856926, -0.566260039806366, 0.03307497128844261, -1.6472426652908325, 0.1842312067747116, 0.9384700059890747, -1.2277116775512695, 0.5093739628791809, -0.1275520920753479, -1.7283642292022705, 0.4188743531703949, -0.19741269946098328, 0.29678285121917725, 1.2682775259017944, -1.1229875087738037, -0.32242971658706665, 0.44622471928596497, 0.4208723306655884, 0.9877258539199829, 0.012169980444014072, 0.4722900688648224, -2.055784225463867, -1.503499984741211, 2.5638601779937744, 0.11886283755302429, 0.14441978931427002, 0.6027865409851074, -0.2044028788805008, 0.5039263367652893, 0.8359526991844177, 1.5588693618774414, 0.9460054636001587], [1.5127147436141968, 0.10923410207033157, -1.5601643323898315, -1.564576506614685, 0.24683883786201477, -1.697163701057434, -1.1076444387435913, -0.28551799058914185, -1.666159987449646, -1.3323142528533936, 1.575332760810852, -2.4248030185699463, 0.42903557419776917, 1.4416590929031372, -0.5669082999229431, 0.8348279595375061, -0.3478472828865051, -1.9715642929077148, -1.3710604906082153, -0.6621795892715454, 0.2524488866329193, 0.8020097613334656, -1.6652801036834717, -0.6097419261932373, -0.2570395767688751, 1.8028762340545654, -0.9212073087692261, -0.7932335734367371, -0.08111102879047394, 2.2831993103027344, 0.7018300890922546, 1.0987281799316406, -0.5531085133552551, 0.8438512086868286, -1.4094514846801758, 0.5796729922294617, -1.0926614999771118, -1.5191199779510498, -0.1379438191652298, 0.4386412501335144, -0.24932612478733063, -0.6025643348693848, -0.9201748967170715, -0.9438930749893188, -0.26826298236846924, -0.7999089956283569, -0.1102561429142952, -0.504916250705719, -0.04493434354662895, -1.0851871967315674, -0.4588196575641632, -0.45752230286598206, -1.054225206375122, 0.8949655294418335, -1.2083197832107544, -0.06859009712934494, 0.330466091632843, -0.527742862701416, 0.8308308124542236, -0.873674213886261, 0.3829450011253357, -0.07462189346551895, 1.800844669342041, 2.794610023498535], [-0.6921586990356445, -0.8503729701042175, 0.4410088062286377, 1.0848658084869385, 0.11094312369823456, 0.7289672493934631, 0.33459144830703735, -1.4259487390518188, -1.6063913106918335, 0.6255906224250793, 0.8307449221611023, 0.6526796817779541, -1.2520606517791748, 0.8027992844581604, -0.8526386618614197, 0.7499589920043945, 1.5145868062973022, -2.5099992752075195, 0.08015192300081253, -0.06182711943984032, 1.1788023710250854, -0.020011618733406067, 0.6625853180885315, 0.20529544353485107, 1.488787293434143, 0.0994483157992363, 0.18943825364112854, -0.9505947232246399, 0.9437848329544067, 0.43566086888313293, -0.7277262210845947, 1.1440589427947998, 0.9426097273826599, -0.5073220729827881, -0.198794424533844, 0.0422251932322979, 0.33264777064323425, 0.24678881466388702, 1.3650163412094116, -0.05700240284204483, 1.3196015357971191, 1.0087125301361084, -0.05564025416970253, 0.8452534675598145, 1.0070271492004395, 0.7665840983390808, -0.43260371685028076, -1.7218382358551025, 0.6666920781135559, -1.129780888557434, -0.07274111360311508, -0.5090555548667908, -0.6198654770851135, -0.4199982285499573, 1.1706024408340454, 1.1131458282470703, -1.2849286794662476, 0.8668932318687439, -1.2379670143127441, 0.7258245944976807, 0.5062667727470398, -0.38412219285964966, 0.18008750677108765, -0.7567539215087891], [-0.3609956204891205, 1.5043669939041138, 0.34667691588401794, 0.14072133600711823, 1.6279066801071167, -0.06041767820715904, 0.25141826272010803, -1.0782743692398071, -0.18094143271446228, -0.0365896113216877, -0.3777073323726654, -0.22631944715976715, -1.4662506580352783, -0.34398651123046875, 1.1675900220870972, -1.2504514455795288, 1.0210223197937012, -0.029138511046767235, -0.6377747058868408, 1.6806578636169434, -0.5835267305374146, -0.44347062706947327, -0.4206784665584564, 1.2277940511703491, 1.2886884212493896, 1.014674186706543, -0.42107465863227844, -0.00749449385330081, -0.9380744099617004, -0.8759279251098633, 0.07432019710540771, -2.163593292236328, -0.48963502049446106, -0.1138591393828392, 0.5672658681869507, 0.9920285940170288, -0.5298365950584412, -0.6125707626342773, -1.5694893598556519, 0.5102517604827881, 0.11238714307546616, 0.795933187007904, 0.21073737740516663, 1.419080376625061, 1.339786410331726, -0.7606099247932434, -0.24859727919101715, 2.205549478530884, 0.9725971817970276, 0.6486656069755554, -1.338695764541626, 0.37876439094543457, 0.2022983431816101, -0.4909529983997345, -0.439677894115448, 0.6442704200744629, -0.5810597538948059, -1.85905921459198, 0.0027132960967719555, 1.675068736076355, 0.10352570563554764, -0.6619934439659119, -1.3441931009292603, 0.25599855184555054], [0.3993191719055176, 1.4167901277542114, 1.4511847496032715, 0.7698419690132141, -1.0330615043640137, -1.280673861503601, 0.5693547129631042, 1.4148510694503784, -0.698388934135437, 0.9132625460624695, 0.9673208594322205, -1.4794764518737793, -0.05972767248749733, -1.6533176898956299, -0.4024379849433899, -0.6799649000167847, -0.7872596979141235, 0.7828943133354187, -0.2452598512172699, -0.22994674742221832, 0.6834805011749268, 0.5966885685920715, -0.19370858371257782, 0.6311655044555664, -0.5441412329673767, 0.578898549079895, 0.020053135231137276, 1.4442721605300903, -1.163649320602417, -0.333526074886322, 1.635486364364624, 0.22160781919956207, 1.6837918758392334, -0.6797250509262085, 0.5380452275276184, 1.0615041255950928, -0.6758207678794861, -0.520061194896698, 1.7688888311386108, -0.9180628061294556, 0.8448523879051208, -0.8811812996864319, -0.9233390688896179, -0.5564686059951782, 1.2376887798309326, 0.2784552574157715, 1.4948272705078125, 0.8946889638900757, -1.1485916376113892, 1.393223762512207, 0.15960295498371124, 0.6854619979858398, 1.101353406906128, -0.7740679383277893, 1.9991728067398071, -0.8991057276725769, -2.0878586769104004, 0.13600318133831024, 0.23460355401039124, -0.9752837419509888, 0.22383226454257965, 0.9563080668449402, 0.21411119401454926, 0.6924965977668762], [0.03585965931415558, -0.2932547926902771, 0.057068102061748505, -0.10496341437101364, 0.2547469139099121, 2.18852162361145, -2.263375997543335, -0.5969652533531189, 0.9704887866973877, -1.0848464965820312, 1.3377307653427124, -0.44404298067092896, -1.2619109153747559, 0.0005504470318555832, 0.7174952030181885, 0.2870131731033325, -0.7918433547019958, -0.4731837213039398, -0.08024735748767853, -0.9835026860237122, -0.39437195658683777, -1.5016330480575562, -0.27164459228515625, 0.20100438594818115, -0.009094967506825924, 0.34264087677001953, 0.7926462292671204, -1.1332310438156128, 0.26971399784088135, 0.7958877682685852, 0.2699744403362274, -0.4925084114074707, 1.0698330402374268, 0.28536900877952576, 1.010688066482544, -0.7751576900482178, 0.0939323753118515, 0.13297994434833527, 1.8500559329986572, 0.9794682264328003, 0.27143552899360657, -2.9359986782073975, -0.9152078032493591, -0.6005725264549255, -0.45580393075942993, 0.5021671652793884, 0.5048781633377075, -1.1001471281051636, -2.370821475982666, -0.481088787317276, -2.021113634109497, 2.1289525032043457, 0.7589777112007141, -0.5421669483184814, -0.2664813697338104, 0.26895198225975037, 0.29366832971572876, -2.606485366821289, 0.3185998797416687, 1.1013987064361572, -1.6067789793014526, 1.6181433200836182, -0.3774340748786926, 1.1313360929489136], [0.05215439200401306, 2.1550960540771484, -0.5763251781463623, -0.282823771238327, -0.3432100713253021, 1.2629122734069824, 0.48558908700942993, 0.3107209801673889, 0.7560486793518066, 0.20298367738723755, 1.9088053703308105, -0.4112128019332886, 1.7255645990371704, 1.408133625984192, -0.8170198798179626, -1.2613871097564697, -0.9330076575279236, 0.6638126373291016, 0.9433552622795105, 0.20536211133003235, -0.9523423910140991, -0.9012035727500916, 0.8726140856742859, -2.0729727745056152, 0.11531700938940048, -0.26718252897262573, -0.7591634392738342, 0.2545230984687805, 0.30288296937942505, -0.7022204995155334, -0.22472980618476868, 0.9406379461288452, 0.22155612707138062, -0.26612886786460876, -0.8377185463905334, 1.735865831375122, 1.6046764850616455, 0.8494120240211487, 0.1231367364525795, 0.5083157420158386, -0.24593479931354523, 0.28543365001678467, -0.17153047025203705, 1.721904993057251, -0.7615042924880981, 0.15535372495651245, 0.37979671359062195, -0.5004294514656067, -2.1819396018981934, -0.027249258011579514, -0.07141538709402084, -1.0675077438354492, -0.49220776557922363, 1.5827436447143555, 0.5958086848258972, 0.02253977209329605, -1.3600677251815796, -1.4665212631225586, 0.9713907241821289, 0.13859228789806366, 1.4411342144012451, -1.9941155910491943, 1.1938196420669556, 1.1223139762878418], [0.9559204578399658, -0.23450694978237152, 1.2845081090927124, 1.622554898262024, -0.8544725179672241, -1.487473964691162, -1.199458360671997, 1.2756060361862183, 2.805706262588501, -0.8984077572822571, 0.7812674045562744, -0.8967714309692383, 0.23747366666793823, 0.33011817932128906, 0.6144965291023254, 1.3668105602264404, 0.7657490968704224, 0.3671908676624298, 1.3768504858016968, -0.9337235689163208, -0.969914972782135, 0.7134606242179871, -0.6915550827980042, -0.4624164402484894, 1.3006656169891357, 0.19616495072841644, -0.7159212231636047, -0.9605036973953247, 1.4829256534576416, 0.5868003368377686, -0.6565295457839966, -0.4708236753940582, -0.5555704832077026, 0.6442517638206482, -2.393242359161377, -0.710811197757721, -0.2116122543811798, 2.832399845123291, -0.6724492311477661, 1.0401848554611206, 0.3579100966453552, -1.0935773849487305, -1.022825002670288, 0.917515218257904, 1.3703835010528564, -0.749238133430481, 0.9827501177787781, 0.631783664226532, 0.7252281308174133, 0.6031719446182251, 0.03669951483607292, 0.24803780019283295, -0.8714985847473145, 0.6240440011024475, 0.07581942528486252, -0.2658647298812866, -0.9909916520118713, -1.6504541635513306, -0.8736976385116577, 0.299459844827652, -1.8517855405807495, -0.8454701900482178, -0.11352746188640594, 1.9115310907363892], [-0.7080115079879761, 1.5958538055419922, -1.5942349433898926, -0.559880256652832, 0.8622316718101501, -0.7349211573600769, 2.894439220428467, -0.3171658217906952, -0.687881350517273, 2.7882161140441895, -0.20010773837566376, -1.7040839195251465, -0.980506181716919, 0.527259111404419, 1.0850555896759033, 0.027920901775360107, 0.6200531125068665, -0.9013931751251221, -0.1389428824186325, -0.7109224200248718, 0.39693683385849, -0.6614686250686646, -1.0951664447784424, 0.049374379217624664, 0.9523376226425171, -0.5924988985061646, 0.08497410267591476, -1.2077537775039673, -0.9335119128227234, -0.590919554233551, 0.8107913732528687, -1.84226655960083, 0.2322145402431488, -0.8765461444854736, -0.12834596633911133, -0.7353071570396423, 0.4294467568397522, 0.23420578241348267, -0.5445629358291626, -0.32176920771598816, 2.193833112716675, -0.5202358961105347, -0.1206074208021164, -0.8061667084693909, -0.3905516266822815, -0.24530187249183655, -0.6477438807487488, 0.26488929986953735, -0.0213184654712677, -1.6922225952148438, -0.2005164921283722, -0.4205782115459442, -0.329731285572052, 0.19425448775291443, -0.0968075692653656, 0.795219361782074, 0.2656405568122864, -0.17905233800411224, -1.128333330154419, 0.13535088300704956, 0.613771378993988, 2.431872606277466, 1.098495602607727, -0.0393231026828289], [2.1570661067962646, 0.2886728048324585, -1.884324550628662, 1.35430109500885, -0.38421210646629333, 0.8491305112838745, -0.33809787034988403, 0.9596397280693054, -1.2938597202301025, -1.077455759048462, 1.2513407468795776, 0.06525728106498718, -0.8480680584907532, -0.21850116550922394, 0.7047998309135437, -0.8223949074745178, 0.6631537675857544, 0.7085955739021301, -0.115847647190094, 1.074539065361023, -0.22356200218200684, -1.0862706899642944, -0.1756971776485443, -0.19138944149017334, -1.0115725994110107, -0.09615857899188995, -1.4263447523117065, -0.3261266052722931, -1.1878526210784912, 0.8259011507034302, 0.9071471691131592, -0.36392998695373535, -0.2768170237541199, 1.576109528541565, 1.219139575958252, -2.043731927871704, -0.07873571664094925, 0.821139931678772, 0.8759386539459229, -0.8594695925712585, -0.04644074663519859, -0.4711352288722992, 1.2245738506317139, 0.7505481243133545, 0.45070767402648926, 1.3054896593093872, -1.4301029443740845, -0.7898744940757751, -0.023410843685269356, -1.5729484558105469, 1.9207477569580078, 0.4920210540294647, 0.23787717521190643, 0.13114024698734283, 0.6596533060073853, -0.17568247020244598, 0.452030211687088, -0.6083998680114746, 0.18105976283550262, -0.08688129484653473, -2.763990640640259, 0.7321276068687439, -0.8336982727050781, -0.41301819682121277], [-0.17384600639343262, 0.2738662362098694, -0.7265743017196655, -2.38682222366333, -0.9336028099060059, -1.0808624029159546, 2.1533517837524414, 0.12181983888149261, -0.9834787249565125, 0.9514744877815247, -0.23821964859962463, -0.40969687700271606, 1.2315864562988281, 0.45569202303886414, -0.7676450610160828, 1.238572120666504, 0.8185160756111145, -0.420842707157135, 0.33993059396743774, 0.8435887098312378, -0.703870415687561, -0.9701611399650574, 0.668861448764801, -0.9539352655410767, -0.5181897282600403, 0.24520441889762878, 1.96187162399292, 1.4436211585998535, 1.0041844844818115, 1.590997576713562, 0.3518112003803253, 0.6799167990684509, 0.8893822431564331, -1.7834035158157349, -1.3634660243988037, -0.36880776286125183, 0.034590646624565125, 1.1708060503005981, 0.7343820333480835, -0.09526888281106949, -1.5056630373001099, -0.8012911081314087, 2.113118886947632, -0.5640999674797058, 1.1804916858673096, 0.7034510374069214, 0.0346762090921402, -1.1723129749298096, -0.8467184901237488, 0.2365599423646927, -0.4916273355484009, 0.40812644362449646, -0.07068563252687454, -0.45779314637184143, 0.0610630139708519, -0.17603962123394012, 0.4235886037349701, 0.12123598903417587, -2.139530897140503, 1.27285897731781, 0.5478617548942566, -0.7291916012763977, -1.0117030143737793, -0.9946556091308594], [-2.5794920921325684, -1.4954336881637573, -0.4832134246826172, 0.0929049551486969, 0.39175331592559814, -1.539650797843933, 0.2952209413051605, -0.784682035446167, -0.8054539561271667, 0.5006036162376404, -0.3525366485118866, -0.27045223116874695, -1.0525579452514648, -0.4676409959793091, 0.7552855014801025, -0.2559644281864166, -1.2738025188446045, -1.7749195098876953, -0.38946983218193054, 0.5498237013816833, -1.3115333318710327, 0.06410734355449677, 0.29154253005981445, -1.6399939060211182, 0.38086554408073425, 0.29818007349967957, -0.4001446068286896, 1.1817970275878906, 0.3995817005634308, -0.7470396161079407, 0.29292526841163635, 0.4207746684551239, 1.3214300870895386, -0.44680777192115784, -2.3591177463531494, 0.026451608166098595, 0.9907160401344299, -1.5746557712554932, 0.6553279161453247, 0.7892070412635803, 1.4433449506759644, 0.9054510593414307, 0.35540062189102173, 1.386415958404541, 0.3361167013645172, -0.742978036403656, 0.7608093023300171, -0.11994092166423798, -0.16605302691459656, -0.3621949851512909, -1.7993487119674683, 0.895347535610199, 0.19348612427711487, -0.9369022846221924, 0.9020295739173889, 0.7601436376571655, 1.3983193635940552, 0.3422759771347046, -0.04856712743639946, -1.1691982746124268, -1.8802968263626099, -0.3781471848487854, -2.3278090953826904, -2.003553628921509], [2.3646345138549805, 0.7374177575111389, 1.4649181365966797, 0.037129901349544525, 0.3073534369468689, 2.0424909591674805, 0.6242370009422302, 0.10331748425960541, -1.0496050119400024, 0.8249403834342957, 0.5559678077697754, -0.6735679507255554, 0.5042310357093811, -0.2092301994562149, -0.1012200191617012, -0.950596272945404, 0.1623416692018509, -1.721390962600708, -0.8078516721725464, 1.9017038345336914, -0.05782260745763779, 2.4866318702697754, -1.1240260601043701, 0.06861645728349686, 1.4329876899719238, 0.16321992874145508, -0.7602483034133911, -1.1389050483703613, -0.9014065861701965, -1.6055043935775757, -0.967433750629425, 1.4630348682403564, 0.2507346272468567, -0.6375497579574585, 0.40769442915916443, -0.1981050968170166, -1.0611711740493774, 2.327937364578247, 0.9429979920387268, -1.1177335977554321, 0.7278094291687012, -1.8470613956451416, 0.7584826946258545, 1.0535401105880737, -0.5368276834487915, 1.6384941339492798, 0.2224503606557846, 0.7855058312416077, 0.5544635653495789, -1.7374650239944458, 0.1442580670118332, -1.312578797340393, -0.45173925161361694, -0.7375869750976562, 0.8021810054779053, 0.391654908657074, 1.0638580322265625, -1.6014811992645264, -1.8574116230010986, 0.18480470776557922, -2.1406075954437256, -0.7367150783538818, -0.04938129708170891, -0.3276449143886566], [-1.4257020950317383, 1.4497425556182861, -2.2347869873046875, -0.8396747708320618, -0.1873052716255188, 1.3593634366989136, 0.7737974524497986, 0.12974993884563446, -0.6770909428596497, -1.0568807125091553, 0.318713903427124, 0.24909153580665588, 0.7960196137428284, 0.6082507371902466, 0.9904279708862305, -1.3040754795074463, -0.2959516644477844, -0.9618374705314636, -0.1568499207496643, -0.6328796148300171, -0.13663606345653534, -1.255192518234253, 0.4047001600265503, -0.3547039031982422, -0.21766096353530884, -0.2578643560409546, -1.1256150007247925, -0.6399197578430176, -0.5009015798568726, -0.8065475821495056, -1.0374828577041626, -0.14399342238903046, -0.5219421982765198, 1.9725816249847412, 0.06591662019491196, 0.05130453407764435, -0.42389002442359924, -0.62079918384552, 0.19953982532024384, -0.8326433300971985, 0.7042095065116882, 0.9085248112678528, 1.69487464427948, 1.1556756496429443, 0.38791415095329285, -3.8662490844726562, -1.525139570236206, -0.6460473537445068, 1.4451301097869873, -1.4830065965652466, 0.8912214636802673, 0.5459232330322266, -0.2044936865568161, -0.5056558847427368, 0.5803678035736084, -1.1411625146865845, -0.5713633298873901, 0.11617931723594666, 0.5947244167327881, -0.9597733020782471, -0.8777239322662354, 2.303459405899048, 0.24590860307216644, -1.0624254941940308], [-0.12731245160102844, 0.2663877010345459, 0.42367297410964966, 0.31376856565475464, 0.3952372968196869, -0.00463151466101408, -0.18645067512989044, 1.4311866760253906, -0.9718242287635803, -1.0217152833938599, -1.312848448753357, -0.9136444330215454, -0.7281755208969116, 1.3098970651626587, -1.1539437770843506, -1.3583033084869385, 1.0639225244522095, 1.207190752029419, 1.0585451126098633, -0.286670982837677, 0.9789793491363525, 0.39648711681365967, -0.8140521049499512, -0.48410677909851074, 0.5884100794792175, -1.4693212509155273, -0.5700135827064514, 1.0499632358551025, -1.1592820882797241, 0.8598187565803528, 0.05505756288766861, -1.1732841730117798, -1.0193883180618286, 0.8196816444396973, -0.10555978119373322, 0.1712210327386856, 0.27630674839019775, -0.008817081339657307, -0.2920748293399811, 0.7338476181030273, -0.6680649518966675, -0.2652435004711151, -1.9373574256896973, 1.4471031427383423, 1.060558795928955, -0.36744800209999084, 1.853927493095398, 0.3752400279045105, -0.2526390552520752, -0.05796312168240547, 2.143080472946167, -0.33616945147514343, -0.5362280011177063, -0.37720435857772827, -1.6027382612228394, -0.02352299354970455, -1.0070637464523315, -1.135380744934082, -0.42356351017951965, 0.7312987446784973, -0.11355005949735641, -0.37088173627853394, -0.7800605893135071, 0.8091778755187988], [0.7679252028465271, -0.28462091088294983, 0.3515043258666992, -0.518675684928894, 0.1351233422756195, -1.309288740158081, 0.2764652669429779, -0.17700523138046265, -1.3211194276809692, 0.9206253290176392, 0.5010265111923218, 0.3172804117202759, 0.007317310199141502, 0.14955982565879822, 0.8432995080947876, -1.1519591808319092, -0.8721807599067688, 0.8093129396438599, -0.5551744103431702, -0.4797220826148987, -1.5197131633758545, -1.1503351926803589, -1.7148056030273438, -0.23413535952568054, -0.16736021637916565, 0.024996845051646233, 0.23496243357658386, 1.0839710235595703, 0.41810330748558044, 0.27947741746902466, 0.23787236213684082, 0.3127502501010895, -0.9179515242576599, 0.2581420838832855, -0.17251229286193848, 0.4276061952114105, 0.18900252878665924, 0.31699469685554504, -0.07328297197818756, -0.7807425260543823, 1.148636817932129, -1.448441982269287, 0.3773588240146637, -0.35345977544784546, -0.31314077973365784, 1.165048360824585, 0.10452961176633835, 0.2347874790430069, -1.016605257987976, -1.5874531269073486, 0.19546474516391754, 0.14653004705905914, 0.8406623601913452, 0.8426821231842041, -0.21857619285583496, 1.7412424087524414, 0.5422226786613464, 0.3291439116001129, 1.0070613622665405, -1.0484024286270142, 0.10950519144535065, -0.05140024796128273, 0.043161120265722275, 0.6011984944343567], [-0.22094258666038513, -1.2197474241256714, 1.0047597885131836, 1.4559568166732788, -1.715131402015686, -0.06726589053869247, 1.1210404634475708, 0.004989977926015854, 0.495785653591156, 0.827812671661377, -1.165136456489563, 0.7303433418273926, -1.3566834926605225, 0.5116437673568726, 1.2283258438110352, -1.7596150636672974, 0.6765739917755127, 0.47546061873435974, 1.6521203517913818, 1.2224407196044922, -1.04555082321167, 0.41915178298950195, -0.06803059577941895, 0.32937103509902954, 1.0688974857330322, -0.8260443210601807, -0.29002684354782104, -1.4741286039352417, -1.4799201488494873, -0.022243507206439972, 0.4529955983161926, -1.4020709991455078, 0.6391648650169373, -1.0182318687438965, -0.9479150772094727, 1.1274081468582153, 0.36732205748558044, -1.6434190273284912, 1.3136790990829468, -0.42129796743392944, -1.5348684787750244, 1.1416226625442505, 2.4122602939605713, -0.18437188863754272, -1.1390786170959473, -0.2699474096298218, -0.8105838298797607, -0.6013668775558472, -0.0012970052193850279, 0.8363082408905029, 1.0151691436767578, -1.6830848455429077, 0.4099046587944031, -0.21534600853919983, -1.8421525955200195, -0.11514446139335632, -0.5998159646987915, -0.6517219543457031, -0.08893990516662598, 0.4383406639099121, -1.0593175888061523, -0.563810408115387, -1.9486761093139648, 2.550096035003662], [1.4841290712356567, -0.09111009538173676, 0.14444506168365479, -0.10890478640794754, -0.4031343460083008, -1.2942780256271362, -0.5442009568214417, -1.77080500125885, -1.1648485660552979, -0.29027414321899414, -3.0023200511932373, -0.49358075857162476, -0.6801393628120422, -1.1134072542190552, 0.15208372473716736, -0.009234224446117878, 0.8076145648956299, -0.2891462445259094, -0.3451938331127167, 1.3948707580566406, -1.483465313911438, -0.5749906897544861, -0.8047899603843689, 0.9742401242256165, -0.9555303454399109, 0.323377400636673, -1.1857880353927612, -2.0025339126586914, -0.039980582892894745, -1.8406907320022583, -1.787868857383728, 0.41923782229423523, -0.7177103161811829, 1.5289260149002075, 0.3728426694869995, 0.50824373960495, -2.541290521621704, -0.6908342242240906, -0.12612707912921906, 0.7595200538635254, -0.8634599447250366, -1.760704755783081, 0.3902553617954254, -0.21823224425315857, -0.13729228079319, -2.8306808471679688, -0.6225894689559937, 1.770153284072876, -0.01327782217413187, -0.29862669110298157, 0.30496737360954285, 0.1559060961008072, 0.13549168407917023, -0.34782513976097107, 0.19218048453330994, -0.6303426623344421, -0.9314399361610413, 0.5965609550476074, 0.03525397554039955, -2.6824512481689453, 0.2917023003101349, 1.23399817943573, 0.21719206869602203, -1.0978926420211792], [0.2747463881969452, -1.4696643352508545, 0.6373141407966614, 0.21575093269348145, 0.14821438491344452, 0.3586481809616089, -0.3953525424003601, -0.29255256056785583, 0.7169448733329773, -1.4536291360855103, -1.6570807695388794, 0.49417394399642944, -0.5069864988327026, -0.12011265009641647, 0.6671191453933716, 1.9868303537368774, -1.0398585796356201, 0.882827877998352, -0.5420464873313904, -0.5884389877319336, 1.2886713743209839, -0.6004747748374939, -0.21008601784706116, 1.033346176147461, -0.3531591296195984, 2.2254655361175537, 1.4563822746276855, 0.4211924076080322, 0.6647191047668457, 0.25872302055358887, 0.3599644899368286, -0.5595983862876892, 0.8590001463890076, 0.09992184489965439, -2.9482216835021973, -2.4164791107177734, -0.614622175693512, -0.47171351313591003, 0.5628142952919006, -0.69048011302948, 0.08691482245922089, -1.2232180833816528, 0.7992574572563171, 0.43489572405815125, 0.12797310948371887, -0.23723670840263367, -0.6508573889732361, -1.4242538213729858, -0.3613484799861908, -2.0943450927734375, 0.43450868129730225, -1.1375397443771362, -0.7610399127006531, 1.4602073431015015, -0.3640950620174408, 0.5654064416885376, 0.8812310695648193, 0.09557853639125824, 0.7917344570159912, 0.7234706878662109, -0.7901054620742798, -0.7130187153816223, -1.2270402908325195, -0.2431143969297409], [1.4706774950027466, -2.475878953933716, -0.24186746776103973, 0.5388603210449219, -0.678055465221405, -0.6146238446235657, -1.2174910306930542, 0.9115258455276489, 0.14770328998565674, -0.8533288240432739, 0.7103821635246277, -0.17862387001514435, -0.45049944519996643, -0.29480427503585815, 0.6961618661880493, -1.1897354125976562, 2.218470573425293, 0.14968009293079376, -0.6576278805732727, -1.325223445892334, 0.28304019570350647, 2.100447416305542, -0.6507156491279602, -1.4698082208633423, -0.7620978355407715, 0.6426225900650024, 1.2973289489746094, -1.0651496648788452, 0.39053210616111755, -0.183901846408844, 1.2395851612091064, 0.6664614081382751, -0.5258887410163879, -0.4839346706867218, 0.35700318217277527, -1.6143430471420288, 2.3767788410186768, -0.9896367192268372, -1.5142383575439453, -0.8219394087791443, -1.9689735174179077, -1.5621485710144043, 1.4551713466644287, 0.32330575585365295, -2.0559170246124268, 0.8662065863609314, -0.654097318649292, 0.7402944564819336, 0.9737029075622559, 0.8519642353057861, -0.06180918589234352, -0.43269091844558716, 0.8885554075241089, -1.1796308755874634, -1.1232850551605225, -1.7537530660629272, 0.26445963978767395, 1.405975341796875, -0.6985955238342285, 0.8138566017150879, 0.3572407364845276, -1.4075521230697632, 0.6472455859184265, -0.3389212191104889], [-0.007210039999336004, 1.4328898191452026, -1.33452570438385, -1.7939026355743408, 0.7483569383621216, -0.3670319616794586, 0.8470219373703003, 0.5368250608444214, 0.4816552400588989, -0.3149580657482147, 0.16140514612197876, -1.9965652227401733, -0.39248254895210266, 0.9412537217140198, 1.301587700843811, 0.17521463334560394, -0.1849755197763443, -0.6072191596031189, -0.2116100937128067, -0.4645753502845764, 0.09406539052724838, 0.1611505001783371, 0.3960413336753845, 0.6019289493560791, -0.5066084265708923, -0.3355740010738373, 0.6084175705909729, 0.776806116104126, -1.9794834852218628, -1.4456799030303955, 1.3322162628173828, -1.3845137357711792, 0.42451924085617065, 0.9436421990394592, 0.030054260045289993, -0.5319281220436096, 1.2970627546310425, 0.0076093547977507114, 0.21985632181167603, 1.2203369140625, -0.31939151883125305, -1.0606458187103271, 1.2028127908706665, -0.6404995918273926, 0.9723179340362549, 1.6515958309173584, -0.4630923867225647, -2.0854928493499756, -0.7546741366386414, -1.0990608930587769, 0.2248126119375229, 0.7474537491798401, 0.003832141635939479, -0.9577687382698059, 1.0826867818832397, 0.45712044835090637, 0.22946906089782715, 0.7416782975196838, 0.31731075048446655, -1.130813479423523, 0.7127590179443359, 2.0157721042633057, 0.9903356432914734, -0.11396408826112747], [-0.9060887098312378, -0.42282435297966003, 1.5583761930465698, 0.6931540966033936, -1.5707879066467285, -0.989512026309967, -0.7934271693229675, -1.0725469589233398, 0.5771760940551758, -0.21396644413471222, -1.6400927305221558, -0.3155273199081421, -0.06769279390573502, 1.9926810264587402, -2.6569607257843018, -0.4411557614803314, 0.6503362059593201, 1.6522656679153442, 0.49952438473701477, -0.4872722625732422, -1.1203269958496094, -1.9124587774276733, -2.156836986541748, 1.0584083795547485, 0.1802792251110077, -2.4843051433563232, -0.47649019956588745, 1.3444490432739258, -0.8310514092445374, 1.6160938739776611, 0.5438369512557983, 0.8590773940086365, -1.315555214881897, -1.1417410373687744, 0.33684465289115906, -1.806774616241455, -0.27452898025512695, -0.6462768316268921, 0.3273850381374359, 1.8987913131713867, 1.0937203168869019, 2.14817476272583, -0.3789328336715698, -0.051268719136714935, 0.9265797138214111, 1.0512959957122803, -0.6005398035049438, -0.12535545229911804, 0.7456743121147156, 2.051800012588501, 0.588067889213562, -0.8901758193969727, 2.0585923194885254, -0.8986180424690247, 1.0970745086669922, -0.14497792720794678, 0.10629863291978836, -1.348641037940979, 1.40969717502594, -0.1596958488225937, -1.3016916513442993, 0.9347525835037231, 0.4688706696033478, 0.30755874514579773], [-0.6737380027770996, 0.9759854078292847, 0.5948416590690613, -1.1730854511260986, 0.08742564916610718, 0.026036983355879784, -1.5414239168167114, -1.8023592233657837, 0.36407044529914856, 0.5687662959098816, -0.7824203968048096, 0.03705377131700516, 0.04596731439232826, -0.7223015427589417, 0.028321312740445137, -0.32600298523902893, -0.8409168124198914, 0.6688699126243591, 0.4031679630279541, -0.5499101281166077, -1.2398872375488281, -1.671837329864502, -0.41527417302131653, 1.7698131799697876, 0.23267848789691925, 0.14700272679328918, -0.43705815076828003, -1.2063425779342651, 2.7517893314361572, 1.626435399055481, 0.588581919670105, 0.2864581346511841, -0.3195863962173462, -1.5325535535812378, 0.40017077326774597, -1.1298086643218994, 0.14774107933044434, 0.07372522354125977, -1.1839900016784668, -1.552828311920166, -0.648331344127655, 0.970836877822876, 0.18650130927562714, -1.100602149963379, 0.5223915576934814, -0.3346998393535614, -0.524364173412323, 0.8938053250312805, -0.027131889015436172, -0.5355402231216431, -0.9085854291915894, -1.9353727102279663, 1.0742146968841553, -2.3800573348999023, -0.611912727355957, 0.7171185612678528, 0.2751080095767975, 1.6453289985656738, 0.45690658688545227, 2.24830961227417, -1.8814970254898071, -0.8915212154388428, -0.10597635060548782, -0.010719092562794685], [0.9116823673248291, -0.41402071714401245, -0.2806067168712616, -0.23506276309490204, -0.17807161808013916, -0.9686658978462219, -1.0159152746200562, -2.219768524169922, -0.9564415216445923, -0.8293007016181946, 0.02724602445960045, -1.2977246046066284, 1.6426517963409424, -1.2214338779449463, -0.3393220901489258, -1.4792450666427612, -0.4780169427394867, 0.02376517280936241, 0.5043861269950867, 0.7328771352767944, -1.884454369544983, -0.43002763390541077, -1.5608946084976196, -0.11944347620010376, -0.41389238834381104, 2.5303778648376465, 0.7016075849533081, -0.2349994033575058, 1.3557466268539429, -0.4550141990184784, -0.5910300016403198, 0.14291611313819885, 0.001934428932145238, -0.9724023342132568, -0.805448055267334, -0.24066254496574402, -0.3255845606327057, -1.4105374813079834, -0.9624523520469666, -0.001650143531151116, 1.4594275951385498, 0.4076511263847351, 1.4338302612304688, -0.4572133719921112, 0.598049521446228, -1.5756102800369263, 0.3346501290798187, -1.4674265384674072, 0.4954787492752075, -0.9436930418014526, -0.5771376490592957, -2.2299654483795166, 0.3368569016456604, 1.1202203035354614, 0.8266674876213074, -1.0914769172668457, -0.6515458822250366, 1.0561672449111938, -0.24735908210277557, -1.3220056295394897, -0.32260140776634216, -0.1795073002576828, -0.21944878995418549, 0.14052261412143707], [-1.1711783409118652, 1.8349032402038574, -0.9561381340026855, -2.013460636138916, 0.1961665153503418, -1.5241618156433105, 0.8328564763069153, -0.07701511681079865, 1.8857500553131104, -2.0487380027770996, -0.5014706254005432, -1.250178575515747, -0.01552307978272438, 0.7644595503807068, -0.40459948778152466, 0.030695399269461632, 0.9659223556518555, 0.06807375699281693, 0.5626087188720703, 0.9867668747901917, 0.7098502516746521, 1.8468801975250244, -1.5797131061553955, -1.8882070779800415, -0.22636577486991882, 1.7270864248275757, -0.7011933922767639, -0.08322714269161224, -0.7818610072135925, -0.4512660503387451, 0.2953949272632599, -0.9327372312545776, -0.7571753859519958, 1.6953665018081665, 0.05742039158940315, 0.6427624225616455, 1.3176333904266357, 1.5057731866836548, -0.43387749791145325, 0.959624171257019, 0.419863760471344, -1.2222892045974731, 1.2309579849243164, -0.2910043001174927, 0.2551168203353882, -2.1152939796447754, -0.7378779053688049, -0.08111074566841125, 1.5056294202804565, 0.6655689477920532, 0.36931371688842773, -0.006870533339679241, -1.364288568496704, 0.8398953080177307, -0.7775176167488098, 0.0281611867249012, -0.37209174036979675, 0.126898854970932, 0.5735008716583252, -0.6300199627876282, 1.3145039081573486, -2.2607157230377197, -0.7612063884735107, 0.579695999622345], [2.4311630725860596, -0.3001357913017273, 0.6510319113731384, -0.6674528121948242, -0.6849910020828247, -1.8944892883300781, -0.5923044681549072, 0.6335856914520264, 0.5540481209754944, 1.3645977973937988, 0.6444818377494812, 0.4425160586833954, -0.5803042054176331, 1.7218668460845947, 0.6598466634750366, -0.17638325691223145, 0.06991355121135712, 1.8816027641296387, 1.3054256439208984, 0.12481536716222763, 0.5512875318527222, 1.709840178489685, -1.0979565382003784, 0.2632344365119934, -2.042729377746582, 0.4010190963745117, -0.4781641662120819, -0.7678966522216797, -0.23441806435585022, -0.4113405644893646, 0.5982087850570679, -1.5794001817703247, 1.3814551830291748, -0.05163963511586189, -0.6175272464752197, 0.3759237229824066, -0.476096510887146, -2.7881317138671875, -0.754183292388916, 1.2303426265716553, -1.3998329639434814, 0.7425496578216553, -0.4482642114162445, -0.08667893707752228, 1.8406258821487427, 0.012708804570138454, -1.1280267238616943, 0.27036771178245544, -0.2545408308506012, -0.5449837446212769, 0.5353515148162842, 0.5673559308052063, 2.075577974319458, -0.5564388036727905, 0.9362351894378662, -0.6829251646995544, 0.15536551177501678, 0.5781860947608948, -1.1594456434249878, -1.028518557548523, 0.03943050280213356, 0.6002200245857239, -1.067932367324829, -0.006346734706312418], [0.02595653012394905, 0.7051181793212891, -0.25139644742012024, -0.8294682502746582, -0.30421072244644165, 0.515365719795227, 0.3972139358520508, 0.1915128380060196, 0.7476951479911804, -0.4107472896575928, 0.18499629199504852, -1.2728830575942993, 0.7194824814796448, -0.4011232256889343, -0.3634259104728699, -0.33720412850379944, 0.715988039970398, -0.38605767488479614, -1.8443450927734375, -0.44944700598716736, -1.6434268951416016, -0.9501597881317139, 0.5527334213256836, -1.8058569431304932, -2.079603672027588, -1.3787014484405518, 0.818074107170105, 0.45758187770843506, -0.4350486099720001, 0.2992224395275116, 1.3498883247375488, 1.8331553936004639, -1.0451858043670654, 1.1111172437667847, 0.13216093182563782, 0.09432031214237213, 0.2659163177013397, -0.06035387143492699, 1.0304917097091675, 1.3333224058151245, -1.0806093215942383, 1.6320666074752808, -0.24586398899555206, -1.442113995552063, 1.011523962020874, -0.2062797099351883, 1.085744023323059, -0.27402713894844055, 1.0082939863204956, -0.3621942698955536, -1.3515479564666748, 0.4323744773864746, -0.022690491750836372, -0.5095013380050659, 0.6367711424827576, 0.5986974835395813, 0.6304376721382141, -0.8649230599403381, -2.023430347442627, 0.9669541716575623, 1.079980492591858, 0.49518147110939026, 0.5023553967475891, 0.7823346257209778], [-0.003790526185184717, -0.053808536380529404, -0.09220918267965317, -0.6119475364685059, -1.050062894821167, 0.471411794424057, -0.3309584856033325, -0.2719935178756714, 0.34288209676742554, 0.7773715257644653, 0.02346125803887844, -1.299968957901001, 0.7942572236061096, -1.2790262699127197, -0.5386407375335693, -1.2309834957122803, -0.4296336770057678, 0.5546562075614929, -2.2845773696899414, 0.9389954209327698, -1.7443130016326904, 1.9685732126235962, -0.10966543108224869, -2.0092296600341797, -0.7387453317642212, 0.3416832685470581, -0.1525723785161972, -1.4035520553588867, -1.1730412244796753, 1.3662245273590088, -1.2465590238571167, -0.9256559610366821, -0.39704230427742004, 0.23096677660942078, -0.017320431768894196, -1.5635433197021484, -0.30478593707084656, 0.5130470991134644, -1.0143309831619263, -0.4871389865875244, -0.9035065174102783, 0.2070259004831314, -1.1677922010421753, -0.06550010293722153, 1.2009027004241943, 0.8984742760658264, 1.0234160423278809, 0.39414510130882263, 0.24887333810329437, -1.2399190664291382, 0.6285629868507385, 0.3582763671875, 0.06853027641773224, -0.3623482286930084, 1.2727159261703491, 0.8800167441368103, -0.7795939445495605, 0.14056673645973206, -0.10204143077135086, -0.7139937877655029, 3.189772367477417, -1.4982712268829346, -0.3687169849872589, 0.009593011811375618], [-0.8024545311927795, 0.13654839992523193, -0.8180676102638245, -0.1889311969280243, 0.16580277681350708, 0.6256498098373413, -1.339768648147583, -0.5876716375350952, 0.5568820834159851, -1.5723514556884766, -0.6782035231590271, 0.9236059784889221, -0.3788723349571228, 0.08577246963977814, 1.345603346824646, -0.49320510029792786, 0.11919734627008438, 1.6702382564544678, -0.4094906449317932, 1.5418387651443481, 1.4972165822982788, 1.033795952796936, 1.4875919818878174, -0.17996759712696075, -0.31603682041168213, -0.8253636360168457, 1.2299648523330688, -1.2874867916107178, 0.32384273409843445, -0.6094822883605957, -1.089150071144104, 1.1388086080551147, -0.7757818698883057, -1.0821406841278076, 1.2468973398208618, -0.7353008389472961, 0.02008640207350254, -0.8662182092666626, 0.532741904258728, -0.6453746557235718, 0.2340821772813797, -1.0999342203140259, 0.469327449798584, -1.3464488983154297, 1.7558931112289429, 0.1214957982301712, -0.12151453644037247, 0.33552664518356323, 1.6781593561172485, -1.425641417503357, -0.10720120370388031, 0.441226065158844, -0.3868156671524048, -0.12533506751060486, -0.30964165925979614, 0.3477430045604706, -1.080323576927185, 0.6846662163734436, -0.651444673538208, -0.7517576217651367, 0.7217931151390076, -0.566368818283081, 1.7217941284179688, -1.2836846113204956], [1.5714502334594727, 0.5418113470077515, 0.09242937713861465, -0.1745937317609787, 0.6206520199775696, -0.16377733647823334, -0.7276095747947693, 0.8776443004608154, 0.2786111831665039, -0.6142274141311646, 1.2160236835479736, 0.03136202320456505, -0.40929144620895386, 1.540365219116211, -0.29013296961784363, -0.1737222969532013, 1.3175091743469238, 0.7753790020942688, 0.34792831540107727, -0.21754924952983856, -1.174539566040039, 0.2789140045642853, -0.3140973150730133, -0.10940273106098175, -1.450000286102295, 0.471706748008728, -0.990007758140564, 0.8675515651702881, 2.2391159534454346, -1.6694797277450562, -1.3658097982406616, 1.222516655921936, -0.5569759011268616, -2.1417460441589355, 0.659203290939331, -0.49943894147872925, 1.7106144428253174, -0.9147999286651611, -0.19525468349456787, -0.8030521273612976, -0.8228508830070496, 0.6877176761627197, 0.7420527935028076, -0.10499892383813858, -1.3385918140411377, -0.10279662907123566, 0.7730525732040405, 0.2822997570037842, 1.3725924491882324, -1.6711678504943848, 1.3952364921569824, 1.1904808282852173, 0.9571017026901245, -0.17481021583080292, 0.7955646514892578, -0.42819222807884216, 0.8103992938995361, -0.44689640402793884, 0.686691164970398, -0.3838001787662506, 1.070785641670227, -0.46835583448410034, -2.392883539199829, 0.6704238653182983], [-0.16890649497509003, 0.4945143163204193, 0.20660458505153656, -0.3857836127281189, 2.0610461235046387, -0.5691609382629395, 2.9841973781585693, 1.2421095371246338, -1.0882084369659424, -2.08547306060791, 0.5411374568939209, -0.6523341536521912, -0.7019744515419006, -0.5487335324287415, 0.5524751543998718, 0.7667479515075684, 1.0644001960754395, -0.996679961681366, -2.02829647064209, 0.22719785571098328, 0.849303662776947, -0.058696381747722626, 0.3185041546821594, -0.44345560669898987, -0.27815982699394226, -0.8201467990875244, 0.04485652595758438, -0.36924606561660767, -1.0931241512298584, 0.29477933049201965, 0.6211760640144348, -0.04192547872662544, 0.7963218688964844, -1.2026703357696533, -0.7387615442276001, -1.2854083776474, 0.019573381170630455, 0.3904109001159668, 0.1394743174314499, 0.16763488948345184, 1.9645189046859741, 0.21851998567581177, -0.05128240957856178, -1.8411248922348022, -1.323183298110962, -0.09187950193881989, -1.0464640855789185, -0.5190594792366028, 0.4557343125343323, -1.2943878173828125, 1.2339574098587036, -1.2880858182907104, 0.3726331889629364, -0.5979945063591003, 1.455432415008545, 0.3896206319332123, 0.01897282898426056, -0.11281534284353256, 0.28389593958854675, 0.7764167189598083, -1.2863644361495972, -1.9393253326416016, 0.7315561175346375, 0.6703018546104431], [0.18600589036941528, -0.08600448817014694, 1.181084156036377, 0.80809086561203, -0.8476921916007996, 1.6780205965042114, 0.6154213547706604, -1.2997232675552368, -0.4263862371444702, 0.7573754787445068, 0.023731617256999016, -1.9568336009979248, -0.22681984305381775, -1.071958065032959, -1.8431768417358398, -1.724640965461731, 1.024949312210083, -0.6924776434898376, 0.029723048210144043, 1.5198856592178345, -0.5547856688499451, 1.4086008071899414, 1.190479040145874, -1.3774831295013428, 1.421951413154602, -1.9090615510940552, -1.662595510482788, 1.179113745689392, 1.3795452117919922, -0.4168228805065155, -0.9241223335266113, -0.9623464345932007, -0.8438895344734192, -0.08111191540956497, -0.33216390013694763, 0.0926380380988121, -0.5914490222930908, -0.11435139924287796, -0.9316725134849548, -0.6156989932060242, -1.036738395690918, 0.8722714781761169, -0.9336125254631042, -0.18497981131076813, 0.4578835368156433, 0.8450298309326172, -0.13025246560573578, 0.6857502460479736, -0.5812846422195435, -0.7740517854690552, -1.0195610523223877, 0.9176844954490662, -0.6326528191566467, -0.4076170027256012, 0.1535750925540924, 1.257425308227539, 0.31468644738197327, 0.9241131544113159, 0.6536897420883179, -0.3740997612476349, 0.7066618800163269, -0.048853304237127304, 0.7105762958526611, -0.2640749216079712], [-0.8471086621284485, -1.0763322114944458, -2.1811091899871826, 0.09741643816232681, 0.44891300797462463, -1.3357000350952148, 0.2237156480550766, 0.20848849415779114, 0.20762121677398682, -0.39749035239219666, 0.03497619554400444, 0.36065930128097534, -0.3623465299606323, -0.9410157203674316, 0.4872034788131714, -0.5669316053390503, -0.1675640344619751, 1.28148353099823, 1.2207425832748413, -0.43037107586860657, -1.8165662288665771, 0.9952617287635803, 0.5772233009338379, 1.5446739196777344, 0.33462417125701904, -2.25614333152771, 0.9053972363471985, -0.08431731909513474, 0.22704215347766876, 0.36599498987197876, -0.08729276061058044, -0.3535037636756897, 0.8487542867660522, -0.5694530010223389, 0.30181413888931274, 0.41700655221939087, 0.7745591998100281, -0.7777354717254639, 0.4718005061149597, 0.7183010578155518, 0.37527549266815186, -0.8914844393730164, 0.8721362948417664, -0.627622127532959, -1.7263151407241821, 0.1470758616924286, -0.20458419620990753, 0.5891020894050598, 0.30988284945487976, -0.8320819139480591, -0.7929192781448364, 1.2133435010910034, 1.2560120820999146, -1.1885818243026733, 0.10866036266088486, 0.7893617749214172, -0.15306951105594635, 2.6450440883636475, 0.357240229845047, 1.70022714138031, -0.6544927954673767, 0.9066141843795776, 1.1049659252166748, -0.9591840505599976], [-0.6286747455596924, -0.6029088497161865, -0.41412094235420227, -1.1685168743133545, -0.3785233497619629, 0.7212836146354675, 0.3125742971897125, 0.2327413111925125, -1.5038774013519287, -0.41642966866493225, -0.8305468559265137, -0.8048530220985413, -0.5665959715843201, 1.4679124355316162, -1.2687424421310425, 1.2070854902267456, -0.07504142820835114, 2.0444419384002686, 0.5553368926048279, 0.38470226526260376, -1.5249122381210327, 0.49950292706489563, 1.0723694562911987, 0.8039156198501587, 1.0962471961975098, -2.6064674854278564, 0.8953585028648376, -1.8348864316940308, 1.0409226417541504, 0.20654520392417908, 1.0598297119140625, 1.3820747137069702, 1.2831231355667114, 0.38501793146133423, -0.8169237375259399, -0.6571497321128845, -1.2474370002746582, -1.7607533931732178, -0.2008415013551712, 1.121287226676941, -0.12466847151517868, -0.5660865902900696, -2.471418619155884, -0.32389527559280396, 0.9966247081756592, -0.7831113338470459, -0.6935639381408691, -0.31771573424339294, 0.47026076912879944, -0.3251325190067291, -2.2776339054107666, 1.0808297395706177, 0.23082274198532104, 0.750407338142395, -0.14217022061347961, 0.3207032084465027, -2.395798683166504, 0.31286001205444336, 0.9382011890411377, -1.3806920051574707, 0.21842074394226074, -0.8748372197151184, 0.576945424079895, 0.26008862257003784], [-0.7690603733062744, -0.2571708858013153, -0.34135928750038147, -0.19254285097122192, -1.39553964138031, -1.736902117729187, -0.22138237953186035, -1.6082355976104736, 0.8161473870277405, -2.4069550037384033, -1.4886982440948486, 0.15815037488937378, -1.36915922164917, -1.6272425651550293, -0.03792864829301834, -0.6633972525596619, 0.05911913141608238, -0.7966346144676208, 0.06197977066040039, 1.6399325132369995, 0.9595727324485779, -0.4820690155029297, 0.49568742513656616, -1.4595850706100464, -2.0861997604370117, -0.701173722743988, 2.489166736602783, 0.6871862411499023, 0.7386689782142639, -0.862356960773468, 0.26104387640953064, -1.0695230960845947, -1.7993533611297607, -1.1018933057785034, 0.45151397585868835, 0.09640958160161972, -0.6563829779624939, -2.261263132095337, -0.47319701313972473, -0.6302700042724609, 1.0969057083129883, -0.23841160535812378, -1.5624548196792603, -0.046026188880205154, 1.4436731338500977, -0.31114518642425537, -0.7869027256965637, 0.44120439887046814, -1.0349698066711426, 0.2585981488227844, -1.1603471040725708, -1.4540481567382812, -1.119130253791809, -0.8447045683860779, 0.6965664029121399, -1.7544448375701904, 0.5218867659568787, -0.5850213766098022, 0.7224310636520386, -1.424323320388794, -1.024043083190918, -0.21756680309772491, -1.5841045379638672, -1.3324464559555054], [-1.6052191257476807, -1.1744391918182373, -0.1314232051372528, -1.2217578887939453, -0.8216584324836731, -0.68000328540802, -0.8878185749053955, 0.07030753046274185, 0.6771792769432068, -0.46616387367248535, 0.7467237114906311, 0.9358930587768555, 0.14045597612857819, 2.620643138885498, -1.0836950540542603, -1.9505378007888794, -0.9654238224029541, -2.029266595840454, 0.23206929862499237, -0.8684197664260864, 0.38093212246894836, 0.25369858741760254, 0.04508674144744873, 1.6419172286987305, -0.31424957513809204, 1.9198521375656128, 0.1384446769952774, -0.5912147164344788, 1.2762408256530762, -0.695638120174408, 1.8689419031143188, 0.3727877736091614, 0.8278030753135681, -0.17475883662700653, 0.901235818862915, 0.19703657925128937, -0.4890308082103729, 0.2560320794582367, 0.46409788727760315, 0.5810210704803467, 0.3259539306163788, -1.4765679836273193, -0.9161431789398193, 0.8461688160896301, -0.7559669613838196, -1.0813547372817993, -1.4966034889221191, 0.6589224934577942, -0.5376275777816772, -0.27294236421585083, 0.38800644874572754, -1.1627588272094727, -0.30083489418029785, -0.374316543340683, 0.0619935505092144, 1.167974591255188, 1.5109201669692993, -1.3590552806854248, 1.9788706302642822, 0.5854468941688538, -0.5144929885864258, -0.46193772554397583, 0.06184990704059601, -1.2716461420059204], [-0.11079195886850357, -0.9199929237365723, -0.584388792514801, 0.8946360349655151, -0.31759580969810486, -0.7657284140586853, 1.216709852218628, 0.42326539754867554, -2.9315567016601562, 0.13108482956886292, -0.5370806455612183, -0.21424546837806702, -1.252119541168213, -0.455582857131958, -1.3536649942398071, -1.0708537101745605, -0.7707707285881042, 0.19825609028339386, -1.04092538356781, 0.39128610491752625, -0.47643962502479553, -2.838202953338623, -1.6438227891921997, 0.5045664310455322, 0.7634323835372925, -1.5635364055633545, 0.3589532971382141, -1.0003206729888916, 0.6234062910079956, 0.4034544825553894, -0.4856725335121155, -0.1871972382068634, 0.7020910382270813, -0.8248093724250793, 0.7394993901252747, -1.2094125747680664, 0.3726349472999573, -0.7855426073074341, -1.4827289581298828, -1.232079267501831, 1.23836088180542, -0.6449825167655945, 0.2691699266433716, 1.001930832862854, 0.8968430757522583, 0.0333486869931221, -0.25080806016921997, -1.5057820081710815, -0.011352133005857468, 0.7743291854858398, -1.6539771556854248, 1.1073940992355347, -0.3990465998649597, -0.0027633490972220898, -1.9749222993850708, -0.372589111328125, -0.6651336550712585, 2.635190725326538, -0.719679594039917, 0.133719801902771, -0.5683512687683105, 0.687588632106781, -0.7534265518188477, 1.3164469003677368], [1.2667099237442017, 0.9365081191062927, 0.31003546714782715, 0.6862303614616394, 0.6653604507446289, 1.006942629814148, -1.1927584409713745, -0.1654585897922516, -1.3983262777328491, 1.5342525243759155, -2.410012722015381, 0.8689903020858765, -3.1571106910705566, -0.08902212977409363, 0.34355250000953674, -0.1670757681131363, -1.0049699544906616, 1.374500036239624, 0.6595422625541687, -1.4615168571472168, 0.11054521799087524, 0.37041163444519043, -0.2287384271621704, 0.4324702024459839, -0.6287673711776733, -0.09859853237867355, -0.09945176541805267, -0.21253816783428192, 0.44382500648498535, 3.082066535949707, 0.8037127256393433, 0.38002049922943115, -1.2935535907745361, 0.8821995854377747, 1.6536732912063599, 0.11439139395952225, 1.1830202341079712, 0.3200637996196747, 0.4250868856906891, 0.3184783458709717, -0.5834915041923523, 0.17668314278125763, 2.4255359172821045, 0.16229312121868134, 0.00897459127008915, 1.7189210653305054, 1.3598244190216064, -1.3142476081848145, -0.9691661596298218, 1.968382716178894, -2.40201473236084, -1.7148492336273193, -0.18628259003162384, 1.7936992645263672, 0.5465142726898193, 0.7305127382278442, 0.42262837290763855, -0.5480572581291199, -0.727724552154541, 0.06015872582793236, -0.6256064176559448, -0.7823416590690613, -2.2058048248291016, -2.088066339492798], [1.0604418516159058, -0.8300740718841553, -0.10528796911239624, 0.7167130708694458, -0.42108193039894104, -1.0748486518859863, 0.18724936246871948, -0.9232780933380127, 0.3289370536804199, -0.7246605753898621, 0.6358447074890137, 2.0074148178100586, -0.08588963001966476, 0.43814319372177124, 1.0490484237670898, 0.661291241645813, 0.07894808799028397, -0.641433835029602, 0.567886233329773, 0.6889312267303467, 1.1046611070632935, -0.1932097226381302, 0.1380644291639328, -1.349586009979248, 0.9058363437652588, -0.3349674642086029, 1.4850636720657349, -1.0169154405593872, -0.9119285941123962, -0.5961868762969971, 0.7251092791557312, 1.539964199066162, 0.15535755455493927, -0.7273485660552979, 1.352231502532959, -0.8972791433334351, -0.3307175636291504, 1.0626568794250488, 0.9814999103546143, -0.5182850360870361, -1.6180845499038696, -0.03173862025141716, -1.1747052669525146, 1.391169786453247, -0.9070401191711426, 1.1143996715545654, -0.4080983102321625, -0.5361413955688477, 0.04717060551047325, 0.39735299348831177, -0.8118286728858948, -1.6715282201766968, -2.425571918487549, 1.5162999629974365, -1.6719247102737427, 0.7213367819786072, -1.7806148529052734, -0.9285635352134705, 0.1735238879919052, 0.7724178433418274, 0.4965841770172119, -0.5659860968589783, -0.5822654366493225, -2.086146831512451], [-0.9325069785118103, 1.4146469831466675, 0.014444670639932156, 0.6855667233467102, -0.18577200174331665, 0.03454223647713661, 0.8284235596656799, -1.2623845338821411, -2.125638246536255, 0.3260814845561981, -0.5320022106170654, 0.04938492551445961, 0.9902052283287048, 0.1681855469942093, -1.9753121137619019, 0.8116884827613831, 0.10181591659784317, 0.24427756667137146, -0.7248098254203796, 0.4484842121601105, 0.3302083909511566, 0.13783866167068481, 2.1235454082489014, 0.591820478439331, -0.32218438386917114, 0.38832759857177734, 0.07100838422775269, -0.6676414608955383, 1.8372247219085693, -1.2088329792022705, -0.18928050994873047, -0.2478848099708557, 0.06803512573242188, -0.04611358046531677, 0.44492921233177185, -0.6593605279922485, 1.4941353797912598, -1.3845596313476562, 1.5637640953063965, -1.2982618808746338, 1.2805821895599365, 1.6227506399154663, -3.138667583465576, 0.23786433041095734, -0.5397348403930664, -0.2522479295730591, 1.6079282760620117, -0.12194675952196121, -1.0241646766662598, -0.6939138174057007, -1.1514742374420166, -0.8014469742774963, -0.035767119377851486, -0.037337612360715866, -0.06748175621032715, -0.820833146572113, -0.757470965385437, 1.0570182800292969, -0.24515295028686523, -0.20523446798324585, -0.12597502768039703, 1.161360263824463, 0.34100982546806335, 0.09444854408502579], [0.4584854543209076, -1.420711874961853, 0.2634906768798828, -1.3079873323440552, -2.7224533557891846, 0.19454731047153473, -0.14055094122886658, 1.1768808364868164, 0.5160212516784668, -1.2994567155838013, -1.1838310956954956, -0.7528581023216248, 0.7505608797073364, -0.5479452013969421, 0.47342729568481445, -1.515196442604065, -2.5132155418395996, -1.2938380241394043, 2.5858187675476074, 0.8947672843933105, -0.42381760478019714, 0.1086081862449646, 1.260750651359558, 0.09863397479057312, 0.5106157064437866, -0.4760016202926636, 0.41779187321662903, -0.8405478000640869, 0.10144032537937164, -0.6183794140815735, -0.7187487483024597, 2.2337026596069336, 1.1022659540176392, -0.13357391953468323, -0.054010968655347824, 0.09260425716638565, 0.5201374888420105, 0.7354792952537537, 0.6388417482376099, -0.7601608037948608, 0.5610396265983582, -0.06509079039096832, -1.167267918586731, -0.9967540502548218, -0.07404615730047226, -0.3766883313655853, 0.021625297144055367, -0.5748241543769836, -0.28982606530189514, 0.6483823657035828, -0.330482542514801, -0.06566727161407471, -0.35791486501693726, -0.9600280523300171, 0.10509608685970306, 0.7629037499427795, 0.22687573730945587, -0.1139005497097969, 0.20295113325119019, -0.6080640554428101, -0.11256245523691177, 1.8252350091934204, 0.6225185394287109, -0.38567233085632324], [-0.6375260353088379, -0.11806514859199524, -0.08769767731428146, -2.59018611907959, -0.7547434568405151, -0.002284794580191374, -0.36255043745040894, -2.1796889305114746, -1.357912302017212, -0.555671751499176, -1.0036293268203735, 0.30868563055992126, 1.3186256885528564, 0.6583144068717957, -1.2023810148239136, 0.019601648673415184, 0.14005029201507568, -0.7360214591026306, -0.3462999165058136, -0.014647983014583588, 1.0953929424285889, -0.6726717948913574, 0.27985742688179016, 1.4044513702392578, 0.16320157051086426, -1.4653079509735107, 1.8143078088760376, -0.3253929913043976, -0.9453659057617188, -1.6500239372253418, 0.7177938222885132, -1.383854627609253, -2.018294095993042, 1.064178466796875, 0.806224524974823, 0.17863504588603973, 0.4864939749240875, -0.08846008032560349, -0.16675138473510742, 0.4390670657157898, 1.1440132856369019, -0.4533294141292572, -0.5830315947532654, -0.060230180621147156, -0.08531027287244797, 0.4751691222190857, 0.1885342299938202, 0.015125978738069534, -0.8222908973693848, 1.6784297227859497, 1.7811030149459839, 0.029427219182252884, 1.0338771343231201, -0.30035892128944397, 1.0097529888153076, -0.6298528909683228, -1.5908408164978027, -0.9133660197257996, -0.49883538484573364, -0.38233160972595215, -0.17621131241321564, 0.9198943972587585, 0.6449651718139648, 1.2857866287231445], [-0.8528351783752441, 0.17334721982479095, 0.255174845457077, 0.4610448181629181, 1.0051687955856323, 0.6907315254211426, 0.2749522030353546, 0.02126617170870304, 0.032203491777181625, 0.3087655305862427, 0.3412141799926758, 0.32073497772216797, 0.16702055931091309, 1.2971450090408325, 0.034682780504226685, -1.4103106260299683, -1.4315683841705322, 1.1613891124725342, -0.5276831984519958, 0.32825952768325806, 0.4167730510234833, -0.7159423828125, -0.6859434247016907, 0.33370524644851685, 0.20369495451450348, -1.0103189945220947, 0.5202121138572693, 0.6553365588188171, -0.9109432101249695, 1.5436084270477295, -1.0894018411636353, -0.2409246414899826, 0.332062304019928, 0.37148794531822205, -0.39986202120780945, -3.2569453716278076, -1.9534255266189575, 0.804378092288971, -0.43345049023628235, -0.1516369879245758, -0.10471804440021515, 1.0125418901443481, 0.5668309926986694, 0.42209839820861816, -1.308510422706604, 1.0100915431976318, -0.26974186301231384, 1.0933220386505127, 1.4114863872528076, 1.0265856981277466, 1.4907705783843994, 0.22049298882484436, 0.3007282316684723, -1.3750107288360596, 1.6886780261993408, 1.6216093301773071, 0.24857451021671295, -0.3284817934036255, -0.3430681824684143, -1.5355881452560425, -0.0784391537308693, -0.17588689923286438, 0.7577976584434509, -0.17671936750411987], [1.081792950630188, 0.9110196232795715, -0.21858513355255127, -0.2333371490240097, -0.42434731125831604, -0.7998950481414795, -0.40308400988578796, -1.5015794038772583, 0.4580554962158203, 0.34237340092658997, 0.9417077302932739, -0.024316484108567238, -1.391671895980835, -0.8170037865638733, -0.6906100511550903, 1.6115800142288208, 1.3409837484359741, -0.9843303561210632, -1.2554575204849243, -2.309009313583374, 1.1895335912704468, 0.8249587416648865, -1.396191120147705, -0.6436415314674377, 0.2973974347114563, 0.887025773525238, 0.09717115759849548, -0.12197630852460861, -1.168366551399231, 1.1924505233764648, 1.1466211080551147, -0.5447920560836792, 0.21050800383090973, 1.434295415878296, 2.53302264213562, -0.8261555433273315, 1.1044933795928955, 0.6959661841392517, 0.9848981499671936, 1.8294790983200073, 0.2924959659576416, -1.9973255395889282, 1.0467090606689453, -0.8405215740203857, -0.863932192325592, -1.619980812072754, -1.1837162971496582, -1.7651203870773315, -0.9545273184776306, -0.8764272332191467, 0.15495699644088745, 0.49819207191467285, 0.11848574131727219, 0.24054881930351257, 0.4793509542942047, 0.021259376779198647, 2.063717842102051, -1.3063656091690063, 1.0241186618804932, -0.05464380234479904, 0.5730866193771362, -0.6796156764030457, -1.4131306409835815, 2.602808713912964], [-1.3095804452896118, 0.8890693187713623, -0.5360977053642273, 0.838262677192688, -1.1800254583358765, 0.24689364433288574, -0.05382435396313667, 0.16908219456672668, -0.764256477355957, 0.6802786588668823, -0.8403104543685913, 0.6669553518295288, 1.727042555809021, -0.041083235293626785, 1.598055124282837, 0.9405306577682495, 0.671841561794281, 0.06802979856729507, -0.6360198259353638, 2.019575834274292, 0.4739063084125519, -0.8528785109519958, 0.8558846116065979, -1.1850829124450684, -0.5852177739143372, -0.5155648589134216, 0.7171655893325806, -0.21705441176891327, -0.5780279040336609, 0.4523780047893524, -0.37778469920158386, 2.1863741874694824, 0.07608999311923981, -0.710753858089447, 0.04430416598916054, 0.01760069467127323, -0.4832490086555481, 0.39968010783195496, -0.24978135526180267, 0.7943359613418579, 0.5892530679702759, 0.48857516050338745, -0.8334831595420837, 0.35103273391723633, -1.5006120204925537, -1.7114403247833252, -0.7710661888122559, -0.21291697025299072, 0.9060245752334595, 1.200584888458252, 0.18622680008411407, -0.35772934556007385, 0.49857065081596375, -0.5451696515083313, -0.21156856417655945, -1.3051438331604004, 0.23468957841396332, -1.0019901990890503, -2.0249826908111572, 0.6441689133644104, 0.9053460955619812, 0.68478924036026, -0.7959244251251221, 0.3069615066051483], [0.6851457953453064, -2.0184543132781982, 1.0046309232711792, 0.3926308751106262, 0.09146676957607269, 0.25277137756347656, 0.3957204818725586, 0.302774041891098, 0.7096238732337952, -0.7786480188369751, 1.2131636142730713, -0.40621739625930786, 0.37035927176475525, 1.095738172531128, 1.2112817764282227, 0.2813408672809601, -0.5910730361938477, 0.8646866083145142, -2.2283005714416504, 1.0663881301879883, -0.3912213444709778, -1.4892613887786865, -0.43818536400794983, 0.4650903642177582, 1.0356940031051636, -0.9394958019256592, 1.1218622922897339, -1.5490175485610962, 0.22600381076335907, 0.030546296387910843, -0.3909102976322174, 0.4589895009994507, 0.04449697583913803, -0.8341253399848938, 0.8823733925819397, -0.6878130435943604, 0.2771945595741272, 1.5712181329727173, -0.634433925151825, -0.8168761730194092, 1.0270580053329468, -0.4659607708454132, 0.2311062067747116, 0.6858806610107422, 0.9113432765007019, -1.1249688863754272, -1.03577721118927, -0.4451366364955902, 1.5609489679336548, -1.145905613899231, -0.05169707536697388, -0.7481833100318909, -0.8043626546859741, 0.3149361312389374, 1.2679182291030884, 0.2588348984718323, -1.2929760217666626, -0.1822815239429474, -2.871359348297119, -0.9819631576538086, -0.5855332612991333, -0.7666037082672119, 0.7016695141792297, -0.22468173503875732], [-1.9404619932174683, 2.174027442932129, -0.6255280375480652, 0.254753440618515, 1.3067115545272827, 0.15531061589717865, -0.612079918384552, -0.9332303404808044, -0.10928411036729813, 0.09709502011537552, -0.22367876768112183, -1.2854334115982056, -1.979242205619812, 0.1747305691242218, -1.7919846773147583, 0.3744778335094452, -0.9546560645103455, 1.2349135875701904, 1.6842782497406006, -0.46124041080474854, -0.5337863564491272, -0.23022021353244781, 0.052271414548158646, 0.5188668966293335, -1.553425669670105, -1.1898659467697144, -0.853020429611206, -1.3339130878448486, 0.27300649881362915, 0.24351006746292114, 1.5299773216247559, -0.32937929034233093, 0.798664391040802, -0.4389983117580414, 0.24570207297801971, -1.5827019214630127, -0.41565051674842834, 1.443830132484436, -1.3043855428695679, 0.06410519778728485, -1.0287559032440186, -0.6160898804664612, -0.8234009146690369, 0.15559974312782288, -0.07121257483959198, 1.73004949092865, 1.1752276420593262, 1.404529333114624, 1.067291498184204, 0.6340752243995667, 0.10491758584976196, -0.3158515393733978, 0.13764458894729614, -1.7137038707733154, 0.7725725173950195, -0.25709256529808044, -0.8465878367424011, -0.860374927520752, 1.544366717338562, -0.47628942131996155, -0.8397179841995239, -1.061498999595642, 0.4901560842990875, -0.44777047634124756], [1.2562257051467896, -1.320101022720337, -0.5478008985519409, 0.9464867115020752, 0.032471902668476105, -0.5527874827384949, -1.545710802078247, 0.811115026473999, -1.0233012437820435, 1.27134108543396, -0.3635134696960449, -0.7632979154586792, 1.0720893144607544, 0.10965105891227722, 0.9835794568061829, -0.840589165687561, 0.3917442858219147, -0.06165545806288719, 1.1624780893325806, -1.5586036443710327, 0.06436794996261597, 0.8037710189819336, 0.13627007603645325, 0.654853343963623, -0.9148007035255432, 0.43702232837677, 1.460627794265747, -0.2622610330581665, 0.9154595732688904, -2.0586397647857666, -2.126347541809082, -0.09117170423269272, 0.5567516088485718, -0.4649384021759033, -0.4614448845386505, 1.0052725076675415, 0.8956450819969177, -0.9727384448051453, 0.44253551959991455, -0.39393505454063416, -2.3344080448150635, 0.9394738674163818, -0.3882722556591034, -0.6572126746177673, -1.246295690536499, -0.9362108111381531, -0.7696272134780884, 1.1654125452041626, 0.43972986936569214, 0.3679574728012085, 0.7629612684249878, 0.14050592482089996, -1.6596163511276245, -0.8966721296310425, 0.6382274627685547, 0.7948376536369324, 1.1234569549560547, -0.9932461380958557, 1.6685900688171387, -0.9317092299461365, -0.6051498651504517, -0.41999995708465576, 0.2710414528846741, -0.9511640667915344], [-0.07314792275428772, 0.22896403074264526, -0.8731785416603088, 0.6189436316490173, -0.8245806694030762, -0.9313771724700928, 0.4999828040599823, 1.160325288772583, -0.6068594455718994, -0.31022149324417114, 1.4437882900238037, 0.44106531143188477, -0.5517252683639526, -0.7336289882659912, 1.1158274412155151, 1.4694515466690063, -0.3598010838031769, -0.2628275752067566, -1.0599051713943481, -0.18721255660057068, 1.1302604675292969, 0.42922523617744446, 0.4293537139892578, 1.5720844268798828, 0.6941326856613159, -0.357212632894516, 1.058364987373352, 0.8676065802574158, -1.4460703134536743, -0.8206530213356018, -0.6577647924423218, 0.0709119364619255, -0.5347141027450562, -0.3769505023956299, 0.45591863989830017, -1.7598927021026611, 1.0046335458755493, -0.021415621042251587, -0.38411346077919006, 0.8582271337509155, 1.9094460010528564, -0.6967397332191467, 0.29456132650375366, -0.5463202595710754, 0.5709887742996216, -1.105176568031311, 0.845894992351532, -1.3593484163284302, 0.36072155833244324, -1.370733380317688, -0.04119270667433739, 0.4680158495903015, -0.048526227474212646, 0.7063909769058228, -1.0486857891082764, 0.13543999195098877, 0.6510087847709656, -1.5794178247451782, 0.6861103773117065, 1.923004388809204, -0.5773963332176208, -0.01101649273186922, -0.9776661992073059, 0.6115273237228394], [0.48966485261917114, -1.3174360990524292, 0.6198016405105591, -0.4277704954147339, 1.759160041809082, -0.7260837554931641, 0.12349767982959747, -1.9237213134765625, 0.35600605607032776, -1.4489766359329224, 0.29025837779045105, 0.447643518447876, 0.5019400715827942, 1.5999000072479248, 0.8267431259155273, 1.3335638046264648, 2.241447925567627, 1.4746724367141724, 0.2997528910636902, -0.5994937419891357, -0.9585371613502502, -1.3125141859054565, 0.35551539063453674, -0.20764459669589996, 1.0717893838882446, 0.1184844896197319, 0.9800008535385132, 1.2916609048843384, 0.24505075812339783, 0.8092517852783203, 0.22429223358631134, -1.46749746799469, -0.6270298957824707, 1.243798017501831, 0.15622135996818542, -0.6001566648483276, -0.17776837944984436, 0.30711010098457336, 0.884939432144165, -0.6527977585792542, 0.5455293655395508, -1.8677287101745605, -1.3281354904174805, 0.9382904767990112, -2.036438226699829, 2.1867053508758545, -1.7095670700073242, -0.08880604803562164, 0.5942768454551697, 1.5607870817184448, 0.43786269426345825, 0.8981207013130188, -0.8132950663566589, 0.8560802936553955, 0.5496492981910706, 0.2985844612121582, 2.138592481613159, -0.3610963225364685, 0.08198092132806778, -0.5291340947151184, 0.3342928886413574, -1.0405248403549194, -1.83098304271698, -0.22143229842185974], [-0.4520183205604553, 0.9584537744522095, 0.42669251561164856, 0.6605315804481506, 0.8517212867736816, 1.5564217567443848, 0.5665454268455505, 0.04091430827975273, 0.03490608558058739, 1.0753473043441772, 0.4553306996822357, 0.5831900238990784, -0.8327523469924927, 0.5035668015480042, -0.05848442390561104, -0.22371305525302887, -0.3889872133731842, -0.7135422229766846, 1.6465933322906494, -0.7268552184104919, -0.4690861701965332, -1.3364113569259644, -0.47955578565597534, -0.42896798253059387, 1.4162200689315796, -0.39448925852775574, 0.19836659729480743, -0.9831458330154419, -0.03560473397374153, 1.8664827346801758, 1.7994952201843262, -1.12909996509552, -0.25696977972984314, 0.2567160427570343, 0.9174820780754089, -1.5441040992736816, -1.830640435218811, 0.4351481795310974, -1.0239181518554688, 0.488691508769989, -1.0386821031570435, -0.48783281445503235, -0.1770324558019638, 0.31969907879829407, 0.5729214549064636, 1.6992522478103638, -0.23458018898963928, 0.8375900983810425, -0.21825389564037323, 0.1701287031173706, 0.700637698173523, 0.010660930536687374, 2.4318947792053223, -1.16102135181427, -1.4996287822723389, 0.6030806303024292, 0.5702959299087524, -0.34050852060317993, -1.7309874296188354, 0.4224526882171631, 0.5427348017692566, 1.9697295427322388, -0.5668658018112183, -1.0471127033233643], [1.4673315286636353, -0.39444372057914734, -1.373087763786316, -0.6732361316680908, -1.383923888206482, -1.2996618747711182, 1.1036664247512817, 0.02044510282576084, -0.5102848410606384, -0.33968597650527954, 0.7682487368583679, -0.8662349581718445, 0.6699686646461487, -0.6105600595474243, -0.6211451888084412, 0.3711360991001129, 0.5588181018829346, -0.5333966612815857, 2.0691614151000977, 0.030334847047924995, 0.19701384007930756, 0.9881174564361572, 0.04027434438467026, -0.6267117261886597, 0.3544973134994507, 1.0223069190979004, 1.0788012742996216, 1.4782085418701172, -2.0735411643981934, -0.5651872754096985, -1.794771671295166, 0.42192405462265015, -0.725886881351471, 0.825442910194397, 0.2626355290412903, -0.1521691530942917, -2.5282602310180664, -0.7475263476371765, 0.11513758450746536, 2.1183300018310547, -0.8664140105247498, -0.3345305323600769, 0.4181157052516937, -0.9159888625144958, -1.2493865489959717, -0.7753664255142212, 0.2136693298816681, 1.8073564767837524, -0.2786022424697876, -0.651726245880127, -0.6029388308525085, 1.2557405233383179, -0.4176492393016815, 1.938185453414917, -1.6406595706939697, -1.2722971439361572, 0.16594767570495605, 0.32111549377441406, 0.6636865139007568, 1.1820379495620728, -0.20378617942333221, 1.0663026571273804, 0.9759387969970703, -0.5879641175270081], [-0.6155403256416321, -0.04870199039578438, 0.9525443911552429, 0.7499943375587463, -1.4817808866500854, 1.0629732608795166, -0.6756926774978638, -1.9154123067855835, 2.0383620262145996, 0.05806884169578552, 1.7882746458053589, 0.6397086381912231, 0.17836342751979828, -0.8749418258666992, 0.81281977891922, 0.3413044214248657, -0.9839386343955994, 0.9524883031845093, 1.447195053100586, 0.05950098857283592, -1.2760429382324219, 0.07824215292930603, -0.6666536927223206, -1.4047162532806396, -0.04043254628777504, 0.10681156069040298, 0.973752498626709, -1.3527413606643677, -1.8303806781768799, 0.3319091796875, -0.5040304660797119, 0.37953656911849976, 0.18041647970676422, -0.036509107798337936, -0.4788838028907776, -1.0999159812927246, -1.813027262687683, -1.0567996501922607, 1.3270773887634277, -0.5993846654891968, -1.4269368648529053, -0.763390302658081, 0.9517887830734253, -0.20330484211444855, 1.155948519706726, -0.3501775860786438, -0.006433242466300726, -0.584959864616394, 0.9316169619560242, -0.5514070987701416, 1.0079985857009888, 1.1839725971221924, -1.0717713832855225, -0.5910018086433411, -1.694474697113037, -0.48379477858543396, -1.7418392896652222, 0.10721855610609055, -1.056519627571106, 0.8894221782684326, -0.08629568666219711, -0.8985689282417297, 1.0016305446624756, 0.1510000377893448], [-0.7427442669868469, 2.246122360229492, 0.5858826637268066, -1.1157227754592896, 0.9861042499542236, -0.17533983290195465, -0.9100379347801208, 1.2188524007797241, 1.7244515419006348, -0.47094565629959106, -0.15023870766162872, 2.2608654499053955, -0.07296096533536911, -0.8224024176597595, -1.2007750272750854, -0.7502749562263489, -1.310253381729126, -0.18305794894695282, 1.9658746719360352, -0.9829274415969849, 0.3480083644390106, 0.3265656530857086, -1.4918396472930908, -0.015397255308926105, 1.025267481803894, 0.41237205266952515, -1.4867092370986938, -0.3008481562137604, 0.39724966883659363, 1.7138502597808838, -1.4318569898605347, 0.05081233009696007, -1.0537439584732056, -0.47120344638824463, -0.7734732627868652, -0.19298331439495087, 0.15545447170734406, -0.27424052357673645, 1.3774363994598389, 0.06435751169919968, 0.2068810611963272, 0.04031188413500786, 0.32696813344955444, 0.11593396961688995, 0.7489807605743408, -1.369856595993042, -1.455008864402771, 0.9861067533493042, 0.6572731137275696, 0.5729941725730896, -0.29654690623283386, 0.38226574659347534, 1.1432430744171143, 0.7419282793998718, 0.1164550632238388, 0.6516401171684265, 0.18585942685604095, -1.356864094734192, 0.027543900534510612, 0.24881362915039062, -0.26801127195358276, 0.506556510925293, -0.7748031616210938, -0.4649074673652649], [1.244370698928833, 0.05407392978668213, 1.4477922916412354, -1.8539339303970337, 0.5335491895675659, 0.9907354116439819, 0.2389979362487793, 0.5357164740562439, 1.3980356454849243, -0.24014729261398315, 1.962510347366333, 0.3956179618835449, 0.3928679823875427, -0.3155321478843689, 0.490862101316452, 1.5713915824890137, 0.9071462154388428, -2.0421507358551025, 0.7534482479095459, 1.6687065362930298, 1.2917062044143677, 0.8922414183616638, 0.3014548718929291, -1.7646454572677612, 0.30399635434150696, -1.0524178743362427, -0.748487114906311, 0.8683805465698242, 0.4831242263317108, -0.8871221542358398, -0.231734499335289, 0.004024882335215807, 0.4859570860862732, 0.6452851891517639, -0.6922149062156677, 0.39697131514549255, 0.8939123153686523, -1.051701545715332, 0.4287146031856537, 2.3432605266571045, 0.5652852654457092, 1.3169045448303223, 0.5400781631469727, -1.0743380784988403, 0.19609546661376953, -0.9441937208175659, 1.8739712238311768, -1.252544641494751, 0.1172507256269455, 1.4856116771697998, 0.0714406669139862, 0.7267492413520813, -1.355809211730957, 1.5280178785324097, 1.1009701490402222, -1.6601800918579102, -0.032710786908864975, 1.0760588645935059, -2.3470113277435303, -0.6661862730979919, -0.6263909339904785, -0.414009153842926, 0.6999130249023438, -0.007144044619053602], [0.01739121973514557, -0.17692816257476807, -0.6934802532196045, -0.1695016473531723, 2.5676610469818115, 0.48796984553337097, -0.4165927469730377, 0.20264378190040588, -1.8048888444900513, -1.6945850849151611, 0.14046324789524078, 1.3552650213241577, 1.4460803270339966, -1.163084626197815, -0.45497608184814453, -1.6334338188171387, -1.1181442737579346, 0.33937570452690125, 0.6119762063026428, 0.5822920203208923, -0.3189105987548828, -0.12164672464132309, 1.235802412033081, 0.11779551208019257, 0.3539137542247772, 0.11840617656707764, 1.4806020259857178, -1.7431280612945557, -0.06801485270261765, 0.42601948976516724, -1.4137729406356812, -2.3044021129608154, 0.0801764726638794, 1.2193037271499634, -0.0961405485868454, 0.35206735134124756, 1.3056994676589966, 0.8451982140541077, -0.77802574634552, -1.522830605506897, 1.2233116626739502, -2.3333680629730225, 0.42651137709617615, -0.9305218458175659, -1.1230697631835938, -0.6326004862785339, 0.5819898247718811, 0.36994168162345886, -0.2473164051771164, -0.21747584640979767, 0.010153647512197495, 0.09039748460054398, -0.8508697152137756, 0.9218707084655762, 1.302535891532898, 1.6118804216384888, -1.5228264331817627, -0.12214501947164536, -2.118342161178589, -0.8284340500831604, 2.0746660232543945, 0.5649139881134033, -1.947008490562439, 0.5739086270332336], [0.01175552885979414, 0.6346770524978638, 1.3644353151321411, -2.6027626991271973, -0.5450338125228882, 1.1196686029434204, 0.25678908824920654, -0.6950237154960632, 0.28773608803749084, 1.8791950941085815, 0.7108566761016846, 0.8868362307548523, -0.1905047446489334, -0.5765089988708496, 1.3794281482696533, -0.46557050943374634, -0.6267586946487427, -0.3712499737739563, -1.9051306247711182, -0.15011020004749298, -0.42444202303886414, -0.12308545410633087, -0.45239606499671936, 0.21224665641784668, 2.5075576305389404, -1.2981079816818237, 0.17507478594779968, -0.1808968037366867, 0.0005312667926773429, 2.642976999282837, -0.7552801370620728, 0.508385956287384, -0.9633318185806274, -1.1826890707015991, 1.2204824686050415, -0.8167459964752197, 2.026486873626709, -0.12902511656284332, -0.35049673914909363, 1.536879301071167, -0.4415501654148102, 0.9604430198669434, 0.5284569263458252, -0.5959588289260864, -2.459965705871582, 1.1756190061569214, 0.17938831448554993, 0.16350792348384857, 0.22418062388896942, -3.174161672592163, -0.8453271389007568, -1.0098298788070679, 0.7677120566368103, 0.34146174788475037, 0.9427248239517212, -1.1669460535049438, -1.8245903253555298, -0.8675364851951599, -0.7735845446586609, 0.1829414814710617, 0.0017801839858293533, 0.10766571015119553, 1.4092562198638916, 0.8167353868484497], [0.00972254853695631, 1.5860153436660767, -1.5172176361083984, 0.21862353384494781, 1.937138557434082, -0.7100232243537903, 0.5397277474403381, 0.8267607688903809, -2.1623694896698, -0.19462475180625916, 0.3978588879108429, -0.2309330254793167, -0.453895628452301, -2.1586005687713623, 1.6834678649902344, -0.3941843807697296, 1.5090229511260986, 1.497294306755066, -1.307613730430603, 0.48386597633361816, 0.47381091117858887, -0.03395047411322594, -1.2260621786117554, 0.3953802287578583, 0.6248204112052917, 0.031130559742450714, -1.9829527139663696, 0.5454522371292114, -0.599097490310669, -0.37101301550865173, -1.3275808095932007, -1.635545015335083, -0.36265379190444946, 1.7593063116073608, -0.6992574334144592, 0.47806262969970703, 2.0361249446868896, 1.3917070627212524, 0.021189846098423004, -0.01360945776104927, -0.3006575405597687, -1.713939905166626, 2.8731846809387207, 1.47149658203125, -0.6207806468009949, 0.3680507242679596, -0.2610344886779785, 0.681560754776001, -0.2635427415370941, -1.2379387617111206, -0.12136237323284149, -0.9181894659996033, -0.7506599426269531, -0.5061823129653931, 1.4960895776748657, 0.9310240745544434, -0.6095024943351746, 1.2042609453201294, -0.31208422780036926, 0.5970675349235535, 2.01625394821167, 0.3355989456176758, -0.4180726706981659, 1.4374560117721558], [1.4018361568450928, -1.3709825277328491, 1.028630256652832, -0.3163795471191406, 0.26123008131980896, -0.20825159549713135, 1.6781092882156372, -0.8358582258224487, 0.7531594038009644, 2.141214609146118, -0.6560615301132202, 0.6293323636054993, -1.6926852464675903, 0.8504858613014221, 1.8751486539840698, 0.2777259051799774, -0.8220356702804565, 0.9058107733726501, 1.4737331867218018, 0.6112178564071655, 0.40882429480552673, -0.3773958086967468, -1.0442404747009277, 0.07282064110040665, -1.8114758729934692, 0.168770894408226, -0.8448737859725952, 0.19491346180438995, -2.1090948581695557, -0.37835490703582764, 1.1465134620666504, -0.59947669506073, 0.9879322052001953, -0.8476181030273438, 1.596479892730713, 0.22018630802631378, 0.8930125832557678, 0.4828431010246277, -0.35428157448768616, 2.5201284885406494, -0.7253658175468445, -2.038328170776367, -0.17835992574691772, -0.2074865847826004, 0.7143834829330444, -0.33475208282470703, -1.2491943836212158, 0.07766561955213547, 0.3416084051132202, 0.3033744990825653, -1.1356984376907349, 0.8985444903373718, -1.3691881895065308, -1.385269045829773, 0.6270155906677246, -1.4808326959609985, -1.3603764772415161, 1.900577425956726, 0.0015320780221372843, -0.5213639140129089, -0.35400840640068054, -0.9445065855979919, -0.6615968346595764, -0.15293264389038086], [0.6584100723266602, 0.7668970227241516, -0.2518962621688843, -1.2110339403152466, -1.5452885627746582, -0.06700956076383591, 1.043209433555603, -1.877359390258789, -0.25193408131599426, 0.4651156961917877, 0.7717437148094177, -1.2770429849624634, 0.39870181679725647, -0.8343971371650696, -1.2206422090530396, -0.3363268971443176, 0.12375489622354507, 1.3593077659606934, -1.112784504890442, -1.2512938976287842, -0.1625169813632965, -0.3473513424396515, 1.5525891780853271, 1.4243874549865723, 1.428132176399231, -0.18009701371192932, 1.4307160377502441, 0.47586023807525635, 0.24736209213733673, -0.668607234954834, 1.0813847780227661, 1.4248372316360474, -1.382387638092041, -1.1913856267929077, -1.4093400239944458, 1.2610777616500854, 0.332440584897995, 0.33496418595314026, 0.8831241726875305, 1.1983388662338257, -1.1564981937408447, 0.327839732170105, 1.9269911050796509, -1.5805500745773315, 0.9771024584770203, 0.366269052028656, 1.8732430934906006, 0.7124330997467041, -1.3164397478103638, -0.40265926718711853, 0.6281610727310181, -0.37111344933509827, -0.24063995480537415, -2.1228113174438477, 1.3830397129058838, 0.46433597803115845, 0.18081173300743103, 0.010774283669888973, -1.0972654819488525, -0.456387460231781, 0.43664273619651794, -1.1039098501205444, 1.195009469985962, 0.9067404270172119], [-1.4583897590637207, -0.2705678939819336, 0.2804360091686249, -0.3733519911766052, -0.016246037557721138, 1.149466872215271, 0.10159510374069214, 0.2961854934692383, -1.2554078102111816, 0.8295849561691284, 0.4144783914089203, -1.1842684745788574, -0.37930554151535034, -0.2530902028083801, -0.7085719108581543, -0.033698517829179764, 0.7638437151908875, -1.152625560760498, -0.527942955493927, 0.8703258037567139, -0.4563790261745453, 0.7046199440956116, -0.33249807357788086, 0.36308789253234863, -0.38692256808280945, -0.9983078241348267, 0.07093533873558044, 0.44311094284057617, -0.1629030704498291, -1.3888893127441406, -0.12282352149486542, -0.23501263558864594, -0.9374409317970276, 0.36077019572257996, -0.8988757133483887, 0.4648610055446625, -0.36644020676612854, 0.021959802135825157, -1.4880419969558716, -0.8993605971336365, -0.4679494798183441, -2.9890384674072266, 1.2632877826690674, 0.5636895298957825, -0.8789445757865906, 1.2437245845794678, -0.630916953086853, -0.5795069932937622, 0.2853711247444153, 0.5316647887229919, 0.3746846318244934, -0.2760828733444214, 1.3507031202316284, 0.6268738508224487, 0.3075336217880249, 1.0425440073013306, -0.43344834446907043, -0.13924184441566467, 1.2733045816421509, 0.5347297787666321, -1.653830885887146, -1.2875428199768066, -0.8341399431228638, 0.4654310643672943], [0.14246933162212372, -0.3666020929813385, -0.4201889932155609, -1.1712342500686646, 0.8625180125236511, 2.0422115325927734, -0.9861053228378296, -0.5223894715309143, 0.5414025783538818, -1.0660213232040405, 0.44256892800331116, 0.5607731938362122, 0.0038369344547390938, 0.6152633428573608, -1.3223187923431396, -0.08719336986541748, 0.6630584597587585, -1.7046630382537842, 0.1907595992088318, -2.4706318378448486, 0.9024109244346619, 0.6481540203094482, 1.36076021194458, 0.2931639552116394, -0.8459791541099548, 2.7669801712036133, -0.7725714445114136, 0.18436673283576965, -0.7213279008865356, 1.16563880443573, -0.09663745015859604, -0.20845262706279755, -2.685762882232666, -0.0003696344210766256, -0.671576738357544, 1.4710731506347656, -0.2048136293888092, 2.4946789741516113, 0.7907328009605408, -0.7662723064422607, 1.3402742147445679, 0.08376562595367432, -0.801491916179657, 0.2881169319152832, 0.9202591776847839, -0.5566519498825073, -0.5533763766288757, 2.012141466140747, -0.19105465710163116, -0.5336021184921265, 0.5681467652320862, -0.5347264409065247, 0.24940362572669983, 0.9272195100784302, -0.13324372470378876, -2.332578182220459, 1.0589450597763062, -1.0233334302902222, -2.3136801719665527, -1.1289595365524292, -0.3538186550140381, 0.22949029505252838, 0.597259521484375, -0.39690807461738586], [-0.8039160370826721, -0.4295366406440735, -2.5179529190063477, -0.058049220591783524, 0.6577097177505493, 0.10054419934749603, 0.8386845588684082, 0.6642116904258728, -0.17050230503082275, 0.10311514139175415, 0.4316823482513428, 0.0018917018314823508, -0.722880482673645, 0.4703599512577057, 1.200301170349121, -0.7494094371795654, -0.0364842526614666, -0.9764763116836548, -1.8191472291946411, 1.231370449066162, 1.35988450050354, -0.21273808181285858, -0.40494540333747864, 0.323273241519928, 1.2844771146774292, 2.463325262069702, -0.2819400131702423, -1.906616449356079, 2.260319232940674, -1.0136579275131226, 1.011239767074585, 0.40801218152046204, -0.5061817169189453, 0.05423375591635704, 0.19771979749202728, 0.5445343852043152, 0.2818152606487274, 0.130241259932518, 0.20608565211296082, -1.0743257999420166, 0.43227049708366394, -0.670941174030304, -0.6932954788208008, 0.37638774514198303, 0.5726802945137024, 0.32838764786720276, -0.3298041820526123, -0.5409918427467346, -1.5360095500946045, -2.4394290447235107, -0.2943914830684662, 0.6947475075721741, -0.5948144793510437, 0.06857743859291077, 0.6227264404296875, -0.19029685854911804, -0.7500899434089661, -1.146866798400879, 0.6909609436988831, -0.20899994671344757, 0.9133627414703369, -0.7509567141532898, -1.719451904296875, -0.20187105238437653], [1.380440354347229, 0.10825904458761215, 0.3236311674118042, 0.4946843087673187, 1.209136962890625, -0.6882268190383911, -1.0564857721328735, -0.7666093707084656, 2.1932737827301025, 1.4117766618728638, -1.3561650514602661, -2.055898904800415, 0.30858153104782104, -0.9161837100982666, -0.17348848283290863, -0.7081138491630554, 1.0458378791809082, 0.07523021101951599, 0.0732044205069542, -0.8566567301750183, -2.435373067855835, 0.003233042312785983, 0.037448491901159286, -0.3920142650604248, -0.8026228547096252, -0.34536999464035034, 1.212833046913147, 0.3605667054653168, 0.5949177742004395, -0.7820079326629639, 1.922728180885315, 0.08244170993566513, 0.5094287991523743, -0.8915557861328125, 1.0965477228164673, 0.5422149300575256, 1.781023383140564, -1.1876509189605713, 2.4861373901367188, 1.0446522235870361, 0.6692818999290466, 0.0019767717458307743, -0.6662945747375488, 0.5607328414916992, -1.4526019096374512, 2.142399787902832, -0.41748669743537903, -1.3838094472885132, -0.9412027597427368, 2.3740103244781494, -0.33032235503196716, 1.6494402885437012, -1.649307131767273, -1.4321540594100952, -0.4161722660064697, -0.2178104668855667, -1.5560952425003052, 1.3220175504684448, -0.21348662674427032, -0.030253130942583084, -0.4580061435699463, 1.3489540815353394, -0.08802658319473267, 0.6553395986557007], [-0.43656057119369507, 1.2017467021942139, -0.38692376017570496, -1.236419916152954, 0.9310299754142761, -1.658239722251892, 1.9682289361953735, 0.6305485367774963, -0.009462444111704826, 0.7680239081382751, -1.7943452596664429, -0.8624791502952576, -2.214385747909546, -0.43783146142959595, 0.16209262609481812, -0.4762510359287262, 1.4110605716705322, 0.5573267936706543, 0.2514200508594513, -0.4268493056297302, -1.0313245058059692, 0.3227490186691284, -0.9370370507240295, -0.32909947633743286, -0.39882585406303406, 1.444812297821045, -1.9730639457702637, -0.6291754841804504, -0.6876057982444763, -0.6613537073135376, -0.21108536422252655, 0.4672940671443939, -1.8533625602722168, -0.24604929983615875, 0.4326559007167816, 0.3036993145942688, -0.4733053147792816, -1.7637418508529663, 1.3267431259155273, 0.2698345184326172, -1.1788437366485596, -0.9635542035102844, -1.2679013013839722, -0.007929010316729546, 0.3586123287677765, -0.15391379594802856, -0.25649160146713257, 0.3089648187160492, -0.12486682832241058, -1.666507601737976, 0.10156269371509552, 0.4468444883823395, 0.7145289778709412, 0.15960878133773804, 2.3374080657958984, 1.889791488647461, 0.6926986575126648, 0.6320699453353882, 0.5922579765319824, 2.103783130645752, 2.074954032897949, -0.17111873626708984, -0.9186417460441589, -1.6997381448745728], [0.08556165546178818, -0.0785103589296341, -0.6827839612960815, -0.506177544593811, -0.5148216485977173, 0.8733178973197937, -1.5487600564956665, 0.25996705889701843, -1.5526849031448364, 0.8713854551315308, 0.8691307902336121, -0.1347527801990509, -1.0007437467575073, -1.8136379718780518, 0.46502768993377686, -0.9514150619506836, -1.243985652923584, -0.7410001754760742, -0.4755191206932068, -0.26621687412261963, -0.6011125445365906, 0.016805870458483696, 0.39931297302246094, -0.9357433915138245, -1.4385242462158203, 0.9881407618522644, 0.25549861788749695, 0.16223719716072083, -1.0538454055786133, 1.3684242963790894, 1.2570899724960327, 0.040451183915138245, -0.9302353858947754, -0.44975438714027405, 1.1950421333312988, 0.6743351817131042, -1.1220725774765015, -0.3680409789085388, -0.30922168493270874, -1.1133044958114624, 0.036487895995378494, 1.2874445915222168, 0.5758540034294128, 0.8734926581382751, 0.3740699589252472, -1.306695580482483, 0.782534658908844, -0.4396524727344513, -1.4992353916168213, -0.6527376174926758, 0.43728169798851013, -0.5036715865135193, -2.4145312309265137, 0.0002681219775695354, 0.18612049520015717, -1.0053024291992188, 0.3900871276855469, 1.0457954406738281, 1.9239273071289062, -1.1286202669143677, -1.8048044443130493, 0.5590071678161621, 0.39368072152137756, -0.07855433225631714], [1.4184298515319824, -0.6589902639389038, -1.0755033493041992, -1.1151649951934814, 3.12648344039917, 0.07294119894504547, 0.14993153512477875, 0.279371976852417, -1.110095739364624, 0.7011259198188782, 0.08631973713636398, 0.36418038606643677, 0.5764581561088562, -0.8098989129066467, 0.6184741854667664, 0.010058600455522537, 0.5199379920959473, -1.0766363143920898, 0.12735863029956818, -1.3745057582855225, 0.5339839458465576, -0.8323053121566772, -0.1395941972732544, 0.33127662539482117, 0.9991593956947327, -1.5050756931304932, -0.9140954613685608, 0.5757055282592773, -0.4837943911552429, 0.046050719916820526, 1.376084566116333, 0.44190141558647156, -0.19116804003715515, -0.16140501201152802, -2.3284595012664795, -1.9436677694320679, -1.5718930959701538, -1.2411301136016846, -0.989632785320282, 0.2430342435836792, 1.8039969205856323, -0.18748611211776733, -0.8619371056556702, 0.27767446637153625, -1.0251970291137695, -0.1502220779657364, 0.4989556670188904, 2.366492986679077, 1.262037754058838, -1.3428689241409302, -1.6097973585128784, -1.7804597616195679, 0.2167035937309265, 0.9810681939125061, -1.4281107187271118, 0.03451919928193092, 1.4550366401672363, 1.0213541984558105, 1.1141290664672852, 1.1402642726898193, 0.4349970817565918, 0.04362928122282028, -1.45448899269104, 1.4575577974319458], [1.8294048309326172, 0.28877320885658264, -0.4682554602622986, 0.9334294199943542, 0.9725703597068787, -0.559917151927948, 0.036916881799697876, -2.4687066078186035, -0.45883864164352417, -0.5439468622207642, -0.12923851609230042, -0.5817644000053406, -1.3767974376678467, 0.14934615790843964, -1.1113835573196411, -1.3520230054855347, -1.825170636177063, -1.1893795728683472, 0.8026232123374939, -1.7841817140579224, -1.0750150680541992, -0.20784559845924377, 0.26218071579933167, 0.20888137817382812, 0.3308824598789215, 0.13106761872768402, -0.22832123935222626, -0.3978552222251892, -0.6580201983451843, 1.5155481100082397, 0.7619125247001648, 0.7840054631233215, 0.5403560400009155, -0.8963062167167664, 0.6400367021560669, -0.1890593022108078, -0.8032568693161011, -0.9629508852958679, -0.91439288854599, -1.6156388521194458, 0.23567460477352142, -0.19277642667293549, -0.1580413281917572, 0.09175387024879456, 2.52790904045105, -1.094277262687683, 0.5072016716003418, 1.2069332599639893, 0.7169923186302185, -1.2818536758422852, -0.15442419052124023, -0.3974902927875519, -1.6469227075576782, -1.4927865266799927, -1.6863943338394165, 0.18284626305103302, 1.5437935590744019, 0.054004449397325516, -1.2275570631027222, -2.339116096496582, -0.9507097005844116, -0.6445711255073547, -0.07300353050231934, -0.025246307253837585], [0.30427730083465576, 2.3953874111175537, 0.5663369297981262, -0.8175246119499207, -0.6471162438392639, -1.0038045644760132, -1.2818877696990967, 0.5841647982597351, -0.1303694099187851, -1.6352765560150146, 0.30869394540786743, 0.19045786559581757, 1.9572608470916748, -2.0392863750457764, 0.27879053354263306, -0.8948021531105042, 0.4001227021217346, 1.4640942811965942, 0.9529585242271423, 0.9001296758651733, 0.2029872089624405, 0.042121388018131256, 0.7370368838310242, -1.0362550020217896, 0.21307706832885742, -1.4245997667312622, 0.4914911985397339, 0.11111874878406525, 0.9705104827880859, 0.09008467942476273, 1.3309119939804077, 0.04542500898241997, 2.1576297283172607, 2.047541618347168, -1.1177533864974976, 1.864359974861145, -1.049987554550171, -1.1696785688400269, 0.3252182602882385, -1.345960021018982, -1.9543575048446655, -0.09372082352638245, -0.47503000497817993, -1.5951828956604004, 0.20054033398628235, -1.1201434135437012, -0.2708592712879181, -0.40544530749320984, -0.473202109336853, 0.010202010162174702, 0.5396928191184998, -0.12945158779621124, 0.4806942641735077, -0.17760993540287018, -1.587455153465271, -0.18996168673038483, 0.7498960494995117, -0.9021873474121094, -0.5599497556686401, -0.23898227512836456, -1.3240426778793335, -0.6864498853683472, -0.3981151580810547, 0.448752760887146], [0.562964141368866, 1.591021180152893, -0.9784163236618042, 1.49577796459198, 0.3008635938167572, 1.1391619443893433, 0.11769626289606094, -1.5492113828659058, -0.3099534809589386, -1.4814940690994263, -0.2517794668674469, -0.10382603108882904, 0.013303807005286217, 1.1820732355117798, -1.284088134765625, 1.762770175933838, 0.34227970242500305, 0.7675057649612427, 0.5340577960014343, 0.7086017727851868, -0.24203866720199585, 2.169145345687866, 1.4129472970962524, 0.31358522176742554, -0.7642419338226318, 0.7457849383354187, 0.13082024455070496, 1.0941262245178223, -0.0861624926328659, -0.4725058376789093, 0.13109678030014038, -0.41463956236839294, -0.07066775858402252, -1.079930305480957, 0.10943524539470673, 0.29810622334480286, 1.2173538208007812, -0.4303925931453705, 0.9797223210334778, 0.3814903795719147, -0.2398015707731247, -0.5947638750076294, -1.03229820728302, -0.5584701895713806, -0.8636258244514465, 1.0706356763839722, 0.7577179074287415, -0.6419041752815247, -0.8976359963417053, 0.658108115196228, -0.18946145474910736, 0.4529877007007599, -0.809393048286438, -1.6991087198257446, -0.06569802016019821, 0.10018202662467957, 0.8929924368858337, 0.08925370126962662, -1.3826113939285278, 0.4447198510169983, -1.475608229637146, -0.001181886182166636, -0.46553269028663635, -0.7408056855201721], [-0.5838593244552612, 0.17586424946784973, 1.1552956104278564, 0.1488964557647705, 0.03983743116259575, 0.5471369624137878, -2.0942940711975098, 1.3058509826660156, -0.38552162051200867, 2.735137701034546, 1.5564818382263184, -0.2808506190776825, 0.11424204707145691, 1.298798680305481, 1.0165961980819702, 0.4129802882671356, -0.6099050641059875, 0.6965717673301697, 0.8389274477958679, -1.0856659412384033, -0.587824821472168, -0.21728451550006866, -1.361317753791809, 1.5608769655227661, 1.39552903175354, -0.165082186460495, 0.5211923718452454, 0.5942730903625488, 1.049828290939331, 0.21843063831329346, 0.15857559442520142, 0.7024204134941101, 1.0152286291122437, 0.7624749541282654, 1.9590322971343994, 0.17223067581653595, -1.8416171073913574, 1.1572760343551636, 1.2850757837295532, -0.9373258948326111, -0.6939919590950012, -1.3771259784698486, 0.44508543610572815, -0.3600526452064514, -0.9448701739311218, -0.5013172030448914, -1.2960129976272583, 0.4643979072570801, -0.9286877512931824, 0.7994309067726135, -1.4397387504577637, -0.37091219425201416, -1.8146907091140747, 0.9459613561630249, -1.2059502601623535, -0.35809165239334106, -1.307450532913208, -0.006770517211407423, -0.3240966796875, 0.17212074995040894, 0.26425132155418396, 0.8628990054130554, 0.6300733089447021, -0.3118620812892914], [-0.10420174896717072, 0.6894628405570984, 0.18332313001155853, -1.0226207971572876, -1.3274856805801392, 0.9566057920455933, -2.067906141281128, -0.7387409210205078, 1.9008082151412964, 1.0414611101150513, -1.6612324714660645, -0.8642992973327637, 0.8959761261940002, 0.8646641373634338, 2.3692729473114014, 0.26635533571243286, -1.227360486984253, -0.1527448147535324, -0.8254162669181824, -0.014567333273589611, 1.7900866270065308, -1.5754637718200684, -0.584225594997406, -0.48657387495040894, 1.0104645490646362, -0.1267431229352951, -1.237395167350769, -0.18952341377735138, 0.22089654207229614, -0.35263365507125854, -0.5269700288772583, 0.5285906791687012, 0.6659175753593445, -0.3490530252456665, -1.3599954843521118, 0.7158070802688599, -0.1442946344614029, 0.29570847749710083, 0.1949816793203354, 1.440011739730835, -2.2281429767608643, 0.7293033599853516, 1.5511945486068726, -0.6784361600875854, 0.6512228846549988, 0.38164204359054565, -0.5701990127563477, 0.616946280002594, -0.4424465596675873, -0.058172374963760376, 1.0666040182113647, 2.0493829250335693, 0.4097067713737488, 0.3565075397491455, 1.5346107482910156, 0.8743200898170471, 0.597701907157898, 1.4854702949523926, 0.7826255559921265, 1.7060550451278687, 1.8572527170181274, -1.152112603187561, 1.0466725826263428, 0.7243689298629761], [0.7626588344573975, 0.687415599822998, -0.11348368972539902, 1.1386414766311646, 0.22118248045444489, 0.41985541582107544, 1.8936468362808228, 0.7691286206245422, 0.030782116577029228, -0.45484068989753723, -1.2589943408966064, 0.5716605186462402, 0.8267691135406494, -2.656886100769043, -0.08915140479803085, -0.4824516475200653, -1.4597196578979492, -0.395952969789505, 0.5591399073600769, 1.1499947309494019, -1.830150842666626, 0.7293856143951416, -0.2793711721897125, 0.9472353458404541, -2.46176815032959, 0.36366555094718933, -0.8198264837265015, 1.2434099912643433, -1.7241628170013428, -0.27332553267478943, 0.4879542291164398, 1.1354460716247559, -0.7379936575889587, 1.7940353155136108, -1.7494529485702515, 0.6138554811477661, -1.0053812265396118, 1.6899162530899048, 0.7817688584327698, 2.2225449085235596, 0.6686108708381653, 0.8573789596557617, 1.4848450422286987, 0.6407433748245239, 0.22633470594882965, -1.2655177116394043, -0.1750594824552536, -0.6887577176094055, 1.2589565515518188, 0.5778602957725525, -0.8277586102485657, 0.7156150937080383, -0.4396213889122009, 0.38215282559394836, -0.4940023422241211, -0.1981976330280304, -1.4187469482421875, -1.628211498260498, 0.2219875007867813, 1.267740249633789, -0.5229837894439697, -1.1154602766036987, -0.1484682410955429, 0.35941141843795776], [0.017584314569830894, -0.06238475814461708, 0.7124134302139282, 0.7251152396202087, -0.8699708580970764, -0.7355784773826599, 0.17181839048862457, 0.6346040964126587, -0.9679217338562012, -0.8381829857826233, -0.31277137994766235, 0.586704671382904, -0.052672576159238815, 0.21955908834934235, -0.948216438293457, -0.29186734557151794, -0.27767467498779297, -0.01853873021900654, -0.7084779739379883, -1.3656773567199707, -1.2628029584884644, 1.8887848854064941, 0.5778735280036926, 0.33307191729545593, -0.5043509006500244, 1.3569029569625854, 0.585634708404541, 0.37833690643310547, -0.01744368113577366, 1.3470581769943237, 2.245898723602295, -1.227158784866333, 0.13437886536121368, -1.3082128763198853, -0.4652761220932007, -0.25231802463531494, -0.2609139680862427, 1.9352538585662842, 1.5049465894699097, -0.7239720821380615, -0.20773839950561523, 1.7204910516738892, 1.0391080379486084, -0.6468095183372498, -1.6759240627288818, -0.3607483208179474, 1.0744290351867676, 0.256909042596817, -0.400313138961792, 1.093459963798523, -0.9074151515960693, -0.02840883657336235, -1.6175159215927124, 1.1794465780258179, -2.1356468200683594, 0.059188470244407654, 0.33741700649261475, 0.584858238697052, -0.16380175948143005, -0.4889538884162903, 1.1819859743118286, 0.8375630974769592, 0.1963944137096405, 0.31823039054870605], [0.6056271195411682, 0.463093638420105, -1.6724210977554321, -0.6220132112503052, -0.31747645139694214, 0.24371115863323212, -0.2989748418331146, 0.5152526497840881, -0.47550415992736816, -1.0494214296340942, -0.5213163495063782, -0.7484838366508484, -0.2746292054653168, 1.057197093963623, -0.8173885941505432, 2.3917624950408936, -0.5678679943084717, 1.6180164813995361, 0.024711746722459793, 0.6659627556800842, -0.20633117854595184, -0.6450820565223694, -0.5177147388458252, -0.13034284114837646, -0.38065752387046814, -3.2354953289031982, -0.49646076560020447, 0.7475547790527344, 0.6960679292678833, -0.5701262354850769, -1.1322563886642456, 0.36173704266548157, 0.28167736530303955, -0.31436482071876526, 0.9690719246864319, -0.9060595631599426, -1.1416369676589966, 0.5803804397583008, -0.04815292730927467, -0.49413642287254333, 0.1370009332895279, 0.3541964292526245, -0.020096756517887115, 0.2762150168418884, -0.9149699211120605, -0.8355790376663208, -1.3328670263290405, -0.6259098649024963, 0.4132106304168701, 0.4010327458381653, -0.47077038884162903, 0.3215005397796631, 0.46246638894081116, 0.06543819606304169, -0.1672443151473999, -1.497240662574768, -0.10647042095661163, 0.7623233199119568, 0.5128383040428162, -0.9426937699317932, -0.8100363612174988, 0.39550232887268066, 0.046586472541093826, -0.7773793339729309], [1.8677191734313965, -0.6330299377441406, -0.8403871655464172, 1.253806471824646, 0.006349262781441212, 0.20032086968421936, -0.9461290240287781, -2.478780746459961, 1.0240000486373901, 0.3638366460800171, 1.431388258934021, -0.5525848865509033, -0.4857497811317444, 0.6330161690711975, 1.5080392360687256, 1.0009069442749023, 0.3735824525356293, 0.5465534925460815, -0.2695206105709076, -1.6195317506790161, -0.4638325273990631, -0.10226277261972427, 0.07611434906721115, -0.6098577380180359, 1.7350984811782837, -0.6551364660263062, 0.15826882421970367, 1.2060441970825195, 0.5727627873420715, 0.6180087327957153, -0.7492204904556274, -0.2180105447769165, -0.7379553318023682, -0.5644997358322144, 1.1014093160629272, -0.7549564838409424, -1.6669703722000122, 0.26219508051872253, 0.4441269338130951, -2.013056755065918, -0.7415447235107422, -1.512215256690979, 2.5487143993377686, 0.02575168013572693, 0.9280388355255127, -1.2257665395736694, -0.021322567015886307, 1.1273789405822754, -1.1355870962142944, 0.7142016291618347, -0.4943823516368866, -0.6288297772407532, 1.4563674926757812, 0.013536541722714901, -1.340296983718872, -0.2364519238471985, -1.5009305477142334, -1.6150988340377808, -0.6732566952705383, -0.274392306804657, -0.002008275128901005, -0.9335595965385437, -0.5903494954109192, -0.7266333699226379], [-1.3450899124145508, 0.981296718120575, -0.4743596315383911, -1.0704044103622437, 0.38416069746017456, 1.528183937072754, -0.7032623291015625, -0.5763190388679504, 0.03233504295349121, -0.6505151391029358, 0.3577772378921509, 0.5616065263748169, -0.6076996326446533, -1.720352053642273, -0.45094236731529236, -0.7768009305000305, 0.6686951518058777, -2.505253553390503, 1.0319311618804932, 0.5550218820571899, 1.074568271636963, 0.21204453706741333, 1.105391263961792, -0.30181339383125305, -0.11936122924089432, 0.6438078284263611, 0.8755828142166138, -0.47812700271606445, -0.3099263310432434, 1.1494591236114502, -0.9584370255470276, -0.29212990403175354, 1.250732660293579, -0.6254077553749084, 0.7591633200645447, 0.5936605930328369, 1.4760470390319824, 0.5665242075920105, 0.8676213622093201, -1.0831619501113892, 1.3402550220489502, -1.3772296905517578, 0.8117551803588867, -1.6326889991760254, -0.30418843030929565, -0.36830464005470276, 2.1547787189483643, 1.9764935970306396, -0.41451430320739746, 0.44783952832221985, -0.1068730354309082, 0.6466502547264099, -0.9787220358848572, 0.06640546768903732, -0.4272196590900421, -0.9579785466194153, -0.7750127911567688, 0.872688889503479, 0.1540197730064392, 1.3082661628723145, -0.029726067557930946, 0.40792641043663025, -0.48014867305755615, -0.2820766270160675], [-0.34927526116371155, 0.7979207038879395, 1.7864725589752197, -0.07989422231912613, 0.34678253531455994, -0.7189469337463379, 0.8505263924598694, -0.34279969334602356, 0.6013745069503784, 0.1325703114271164, 0.4499690532684326, 0.09824322164058685, -1.1931699514389038, -0.41587212681770325, -0.973118007183075, -0.8279690742492676, -0.2411300539970398, -1.7447795867919922, 0.12233278900384903, 0.3266426920890808, 0.7366036176681519, 0.5825851559638977, -0.4858509302139282, 0.26689738035202026, 0.7367600202560425, -1.1559029817581177, 0.98171466588974, 1.1885582208633423, 0.03578464314341545, 0.6510661244392395, -2.057271957397461, 0.23520344495773315, 1.1772974729537964, 1.460809588432312, 1.2056729793548584, -1.0008318424224854, 0.5137665271759033, 1.1732698678970337, -1.554207682609558, -2.129926919937134, 0.1446617841720581, 0.4237980246543884, -0.1821540743112564, 0.956778883934021, -0.40605223178863525, 1.5685921907424927, -1.481377124786377, 0.026431869715452194, 2.3613462448120117, 0.837629497051239, 0.04253726080060005, -0.8771169781684875, 0.4837915003299713, 1.7745468616485596, 0.41919946670532227, -1.8199750185012817, 1.372788906097412, -1.1446149349212646, -1.894699215888977, -2.2192399501800537, 0.028354398906230927, -0.6933942437171936, -1.4151298999786377, 0.5866376161575317], [0.0042937640100717545, -2.375109910964966, 1.8480653762817383, -1.0765080451965332, -1.1520905494689941, 0.43216437101364136, -0.594000518321991, -1.1235195398330688, -1.128301978111267, -0.24610909819602966, -0.6058820486068726, -0.9018467664718628, 1.5252246856689453, -0.2447219341993332, -0.2697557210922241, -0.8062502145767212, -0.7560099363327026, 0.2299635410308838, -0.27077367901802063, 1.482174277305603, 2.194821357727051, 0.7461038827896118, 1.728760838508606, 0.28126004338264465, 0.991144061088562, 1.3535932302474976, 2.4487249851226807, -0.5474150776863098, -1.8114590644836426, -0.8962641358375549, 2.5009570121765137, 0.1886671632528305, 1.9355336427688599, -0.11268723756074905, 0.36832430958747864, 2.71455454826355, 0.563556432723999, -1.2535512447357178, -0.13900035619735718, -1.752698302268982, -0.026028435677289963, 0.21165667474269867, 0.3500153422355652, 1.7577284574508667, 0.6703706979751587, -0.8371738791465759, 0.8120708465576172, -1.3454574346542358, 0.05823945999145508, -0.7749029994010925, 1.0863112211227417, 1.3234152793884277, -1.2076172828674316, 0.7017759084701538, 0.09303658455610275, 1.1276671886444092, 0.1759345382452011, 1.0808137655258179, 0.12764866650104523, 0.06531096249818802, 2.449964761734009, -0.5856192708015442, 0.3982347249984741, 0.44501993060112], [2.0300374031066895, 0.3839350938796997, 1.387929916381836, -1.1162840127944946, -0.7779125571250916, 0.5427753925323486, -1.0080273151397705, 0.5966311693191528, 1.5361478328704834, 1.688697099685669, 0.16951949894428253, -2.279398202896118, 1.9870054721832275, 0.9297535419464111, -0.37634119391441345, -0.5796073079109192, -1.9930146932601929, -1.3697409629821777, 0.6231133341789246, 1.5872761011123657, -1.8404624462127686, 1.6308954954147339, -0.7809408903121948, 0.4767695665359497, 0.8209711313247681, -1.7416136264801025, -1.5911316871643066, -0.9330378770828247, 1.4727482795715332, -0.8585204482078552, -0.5637169480323792, 0.45833560824394226, -0.1445561796426773, -0.12592682242393494, 0.009525216184556484, -0.3083380460739136, -1.0501011610031128, -0.1652383804321289, 0.1306571215391159, 0.3761969804763794, -1.4353742599487305, 0.23717576265335083, -1.9042081832885742, 0.9514032602310181, 0.540152907371521, 0.30050745606422424, -0.5184152126312256, 0.378685861825943, -0.8977972865104675, 1.3336790800094604, 0.17571280896663666, -0.9090893268585205, -0.8031538128852844, 0.72124844789505, -0.7622933387756348, 0.20301762223243713, 0.954693078994751, 1.029988169670105, 1.5177534818649292, -0.7804734110832214, 0.8538354635238647, 1.3868205547332764, -0.45832958817481995, 0.2567324638366699], [-0.23287953436374664, -1.248787760734558, -0.20163440704345703, -1.1961504220962524, 0.5716477036476135, 0.5870038866996765, 1.1611372232437134, 0.7895665764808655, -0.27415141463279724, 0.6612585783004761, -0.40533336997032166, 1.4705464839935303, -0.08052265644073486, -0.3084115982055664, 0.2927042543888092, 1.148430585861206, 1.3307238817214966, -0.4019505977630615, 1.1272499561309814, -1.311599850654602, 0.2410912811756134, 1.21135413646698, 0.21252600848674774, -0.8329525589942932, 0.4573160409927368, 0.525859534740448, 1.223032832145691, -0.8713048696517944, 0.5202834010124207, 0.6890528202056885, 1.109590768814087, -0.418140172958374, -0.13846223056316376, -0.2957475781440735, 0.5892758369445801, -0.31742867827415466, 0.24814733862876892, 0.6209171414375305, 0.647281289100647, 0.12330745160579681, 1.588978886604309, 1.388068675994873, 0.8041833639144897, 0.0036735893227159977, 0.531660795211792, 0.4838705062866211, -0.07028144598007202, -0.24279281497001648, 1.1342799663543701, 0.30669543147087097, -1.4282734394073486, 1.4010804891586304, -0.1268346756696701, 1.147874355316162, 1.0505881309509277, -1.1658365726470947, 0.3436988890171051, 0.052195385098457336, 0.11753137409687042, -0.9664640426635742, -0.9716734290122986, -1.058872103691101, -0.3169194757938385, -0.5050991177558899], [-2.4966533184051514, 0.25510236620903015, 1.9798173904418945, -0.30568888783454895, -1.5006047487258911, 1.0218089818954468, -0.14956003427505493, -1.9064562320709229, 0.6573079228401184, -0.6648842096328735, -0.3879847824573517, -1.924142837524414, 1.1253609657287598, -0.17771928012371063, -2.8746984004974365, -0.40645521879196167, -0.8153802156448364, -0.49106886982917786, 0.5556100606918335, -0.6456713676452637, 1.2433303594589233, 0.7927350401878357, -0.30866244435310364, 0.5948272347450256, -0.7278779745101929, 0.7348564863204956, -1.7244895696640015, -1.1827694177627563, -0.05409448593854904, 2.244621753692627, -1.595091462135315, 0.10520249605178833, -0.11542913317680359, 0.34192514419555664, 0.3211270272731781, 0.6657548546791077, -0.0115730632096529, -0.3277861773967743, -0.12168212234973907, 2.098808765411377, -0.2089531272649765, -0.12576615810394287, 0.6063429713249207, -0.39203330874443054, -0.3095197379589081, 0.266236275434494, 2.1995606422424316, 1.096696376800537, -0.463923841714859, 0.047776296734809875, 0.8425222039222717, 0.056335046887397766, -0.19890297949314117, 0.6293128728866577, -1.5318844318389893, 0.045901454985141754, 0.26905667781829834, 0.5739648342132568, -1.6237107515335083, 0.6240930557250977, 0.3192174732685089, -0.34137824177742004, -0.6046103239059448, 0.6832541227340698], [-1.3607864379882812, -1.2886996269226074, 2.776970386505127, -1.3162060976028442, 0.2103058397769928, 0.18525807559490204, -0.5637431144714355, -0.9927400946617126, -0.4692533612251282, -0.6060174703598022, -0.2757677137851715, -0.9142293930053711, -1.5726776123046875, 0.1247035339474678, 0.6694346070289612, 1.6983519792556763, 0.8040265440940857, -0.03489333391189575, 1.534218668937683, -1.152733325958252, -0.5445355176925659, -0.5872280597686768, 0.14239108562469482, 0.1063770055770874, 0.15091153979301453, 0.17419324815273285, -0.5291653871536255, -0.11118766665458679, -0.39702287316322327, -0.4967575967311859, 0.1953076422214508, -0.2894027531147003, 0.30512839555740356, 0.3827286958694458, -0.3141155242919922, -1.2964215278625488, 1.4308654069900513, 0.36495622992515564, 0.5580876469612122, -0.7405098676681519, -0.6217169165611267, 0.1847144067287445, 0.3593291640281677, 1.551759123802185, 0.9104708433151245, 0.37356603145599365, -1.1670430898666382, 0.3737991452217102, 0.24186599254608154, 1.0857069492340088, 1.8409314155578613, 1.4053750038146973, -1.1104456186294556, -0.661760151386261, -0.7732235193252563, 1.6916282176971436, 1.6497807502746582, 2.3981246948242188, 1.0494723320007324, 1.2512110471725464, -0.7621088624000549, 0.11545967310667038, 0.6489962339401245, -0.2185487151145935], [-1.062212347984314, -0.6060630083084106, 0.17577049136161804, 0.7215668559074402, 0.5820214748382568, -0.852552592754364, 0.8525628447532654, 0.6869592070579529, -0.8635573387145996, 0.17234863340854645, -0.3255550265312195, -1.2715336084365845, -0.13500043749809265, 0.04691871628165245, 0.2033139169216156, 0.9910907745361328, 0.18265484273433685, -1.0197442770004272, -0.006565133109688759, 0.14600789546966553, -0.8537659645080566, 0.8568860292434692, 0.13600081205368042, 0.3803447484970093, -1.3774935007095337, 0.7024956345558167, 0.25152695178985596, -0.409740149974823, -0.37502557039260864, 0.04132012277841568, 1.050572156906128, -1.4133565425872803, 0.05915692448616028, 0.3372981548309326, -0.9581108093261719, -0.7758608460426331, 0.4399690628051758, 0.07782085984945297, 0.1741933822631836, 0.859825849533081, 0.48935770988464355, -1.6711759567260742, -0.12461408227682114, -1.9859477281570435, -0.8534488081932068, 1.8846027851104736, 0.014196831732988358, 0.6574950218200684, 1.3847019672393799, -2.0505969524383545, 0.15912847220897675, -0.4581861197948456, 0.4766019880771637, 0.4536062777042389, 0.1626858413219452, -0.8338314890861511, 0.30681681632995605, 0.9781305193901062, -0.31831321120262146, 0.12912814319133759, -1.1945480108261108, -1.290777325630188, 1.4132080078125, 0.75914067029953], [0.007398839108645916, 0.27563104033470154, -1.9427233934402466, 1.3041622638702393, -0.12259142100811005, 0.47652962803840637, -0.4543016850948334, -0.3949412405490875, 0.07483948767185211, 0.16308379173278809, 0.938425600528717, -0.4522378444671631, 1.1977993249893188, 1.1887619495391846, -1.610172152519226, -1.221778392791748, 1.8417563438415527, 0.038544248789548874, -1.8103739023208618, -0.8122187852859497, -0.8574603796005249, 0.4749026894569397, 1.471919298171997, -0.7517080307006836, 0.3998848795890808, 1.7595469951629639, 0.8884076476097107, 0.14327684044837952, -0.9581319689750671, -0.8863520622253418, -0.6196189522743225, -0.0778573751449585, -0.15682409703731537, -0.8241299986839294, -1.6110281944274902, -0.1137220561504364, -0.6088743209838867, -1.24480140209198, -0.3764125108718872, -1.4743905067443848, 0.9206827282905579, 1.1343376636505127, -0.7038330435752869, -0.4225757122039795, 0.556340217590332, 0.7180160284042358, -0.384466290473938, 1.3911305665969849, 0.4413147270679474, 1.6192935705184937, -1.1559791564941406, 0.3976458013057709, 0.9250184893608093, 0.3297620713710785, 0.40100234746932983, -1.081972599029541, 0.7425883412361145, 0.738396942615509, 0.176863893866539, 0.07967089861631393, -0.45703163743019104, 0.7515930533409119, -0.5422556400299072, 0.9413283467292786], [-0.6172118186950684, 1.3032920360565186, 0.13076281547546387, 0.3383307158946991, 0.857367992401123, -0.9831250905990601, -0.15291620790958405, -1.0360007286071777, 0.04679035022854805, 1.6492469310760498, 0.21242311596870422, -0.8000541925430298, -1.123394250869751, -0.28556543588638306, -0.2816125750541687, 0.9538917541503906, -0.25314220786094666, 0.7416561841964722, 0.744472324848175, 1.3615506887435913, 0.366484671831131, 0.34290069341659546, -0.42804086208343506, 0.2304510623216629, -1.944677472114563, 0.8454917073249817, 0.17911355197429657, 1.4780157804489136, 0.5097470283508301, -0.31714460253715515, 0.9335018396377563, -0.4178946912288666, -1.0554934740066528, 0.9574659466743469, -0.5241734385490417, 0.5551983714103699, 0.6146439909934998, 1.1162465810775757, -0.6447643041610718, 0.15646827220916748, 1.5550999641418457, -0.2772073447704315, 0.17305690050125122, 0.1419331133365631, 0.5298764109611511, 0.5280739665031433, -0.23126739263534546, 1.1613446474075317, 1.3080848455429077, 1.1575278043746948, 0.37647244334220886, -1.3530198335647583, -1.010391116142273, -0.25626975297927856, -0.9135553240776062, -1.4944864511489868, -0.49429088830947876, -0.24257497489452362, 0.4188121259212494, -0.11102832853794098, 1.075873851776123, 1.2736506462097168, -0.6250147223472595, -0.3620195984840393], [-1.1694109439849854, 0.1958654224872589, -1.332809329032898, 0.46491512656211853, -1.391721487045288, 0.25900664925575256, 1.0557479858398438, -0.37406861782073975, 1.2768462896347046, -0.8543492555618286, 1.1252658367156982, 1.8528081178665161, 1.1854934692382812, -1.309970498085022, -1.2708609104156494, -0.21587535738945007, -1.449595332145691, 0.1920946091413498, 0.09568619728088379, -0.6574819087982178, 0.2987324297428131, -0.7088938355445862, 1.4006602764129639, -1.5565553903579712, -0.5665769577026367, -0.6548826694488525, -0.0358293354511261, 0.18156632781028748, -0.12255847454071045, 2.2073981761932373, -0.21020469069480896, -1.3351744413375854, 1.4738597869873047, -0.19252175092697144, 1.0118114948272705, -0.49757006764411926, 0.29203376173973083, 0.6747888922691345, 1.2921020984649658, -0.2298409342765808, -1.29505455493927, -0.10887889564037323, 0.1766403168439865, 0.3406991958618164, 0.8276427984237671, 1.804460048675537, -0.033476874232292175, 1.517859935760498, 0.9798409938812256, -0.6842489242553711, 1.1683003902435303, 1.6978724002838135, -0.12864843010902405, 0.4195960760116577, -0.006412115413695574, -0.296261191368103, -0.6636171340942383, 0.0385669507086277, -0.11439916491508484, 0.027146976441144943, -1.4782480001449585, -1.0740092992782593, 0.5742219090461731, 2.1847212314605713], [-0.07836923748254776, -0.7884595990180969, -0.5550526976585388, -1.060585856437683, 0.5940539240837097, 0.9451797604560852, -0.7048578858375549, -0.1480177342891693, 0.3294225335121155, -1.2979896068572998, 0.3502602279186249, -0.022655686363577843, -0.7265332937240601, -0.3483693599700928, -1.1852957010269165, -0.38861745595932007, 0.24438856542110443, 0.4450668692588806, 0.8448247313499451, 1.0180827379226685, -1.0400985479354858, -0.34167930483818054, 0.42836660146713257, -0.12562797963619232, -0.0665246993303299, 1.568142056465149, 0.4569310247898102, -1.527083396911621, 1.0414955615997314, 0.330789178609848, 1.282549262046814, -0.037371572107076645, -0.13177719712257385, 0.7935517430305481, -0.5244265794754028, 1.4234893321990967, 0.2240358591079712, -1.5928350687026978, 1.4696967601776123, 0.5720691084861755, -0.7557348608970642, -0.13873280584812164, 2.2201309204101562, -0.9031895399093628, -0.004423134494572878, 1.2660056352615356, 0.041014671325683594, -1.676345705986023, -0.05709744989871979, -1.082713007926941, -0.12316770106554031, 0.2087487429380417, -0.0049696690402925014, 1.5170316696166992, -0.18737196922302246, -0.5524110198020935, 0.5830042362213135, 0.6567727327346802, 0.7325674891471863, 0.28507861495018005, -0.023018287494778633, -1.9552351236343384, 0.17920012772083282, 0.3470936119556427], [0.044540371745824814, 1.1537656784057617, 0.7686847448348999, 0.5579578280448914, 0.9260116815567017, -0.5074717998504639, 1.9968957901000977, 1.6552538871765137, 0.13900615274906158, 0.25558000802993774, -0.7145050168037415, -0.635643720626831, 0.5410622954368591, -0.8004774451255798, 2.0710501670837402, -1.2773934602737427, -1.2754632234573364, 1.0950888395309448, 0.16427923738956451, 1.2829846143722534, -0.7480362057685852, -0.6940435767173767, -1.002892017364502, 0.44957610964775085, -1.0103137493133545, -1.063455581665039, -0.3731412887573242, 0.41294506192207336, -0.8814566135406494, 0.4400838017463684, 1.1081476211547852, -1.3109592199325562, 0.33060094714164734, 0.049274273216724396, 0.3696884512901306, 0.2653281092643738, 0.09042371809482574, -1.242557406425476, -0.08755040913820267, 0.33328914642333984, 1.1576168537139893, 1.0119479894638062, 0.9555693864822388, 0.0448743999004364, -1.017160177230835, -1.4125171899795532, -1.4188343286514282, 2.049182653427124, -0.038244474679231644, 1.201891303062439, 0.49060848355293274, 0.25277090072631836, -0.5929226279258728, 0.8884353041648865, -1.2294944524765015, 0.43925780057907104, -1.8479111194610596, -1.419063925743103, 0.10614877194166183, -0.9960552453994751, -0.6356953978538513, 1.4158844947814941, -3.052922010421753, 0.30631715059280396], [0.9727568030357361, 0.9579065442085266, 1.0062458515167236, -1.0647013187408447, -0.8275327086448669, -0.9003999829292297, 2.1422762870788574, 1.182253360748291, 0.5891382098197937, 0.3590700030326843, 3.5462558269500732, -0.038518764078617096, 1.8127398490905762, 0.09066107124090195, 1.1385270357131958, 0.9888581037521362, -0.18300305306911469, -1.2407768964767456, -1.3481947183609009, -0.5372157692909241, -2.495746612548828, -0.641189455986023, -1.98970365524292, -1.2331334352493286, 0.35214537382125854, -0.8696934580802917, -0.048570480197668076, -1.011451244354248, -0.3504851460456848, 0.31112271547317505, -0.27679702639579773, 0.5553950071334839, -1.2852835655212402, -0.16916854679584503, 0.28807923197746277, -1.5783571004867554, 1.856528401374817, -0.5946454405784607, 0.20587018132209778, 0.36046379804611206, -0.445950448513031, -0.6093077659606934, -1.3287535905838013, 1.1147844791412354, -0.5032780766487122, -0.9636255502700806, 0.47792673110961914, -0.06521694362163544, 0.4330761730670929, 0.5225229263305664, 0.16128626465797424, -0.4379313886165619, 0.16380076110363007, -0.29707229137420654, -0.8090147972106934, -1.408036470413208, -0.6624493598937988, 0.03898775950074196, -2.486976146697998, -0.02687285840511322, 0.7289944887161255, -0.20664605498313904, -1.0144816637039185, -0.4757002592086792], [0.5140361189842224, -0.8731500506401062, -0.3025115132331848, 0.6851522922515869, 1.129026174545288, -0.7225147485733032, -0.21281495690345764, 0.6099705696105957, -0.19996438920497894, 0.044913485646247864, 0.11312687397003174, -1.5072563886642456, 0.499493271112442, -1.1456631422042847, -1.6298259496688843, -0.458134263753891, -0.029328368604183197, -2.0304205417633057, 0.06055472791194916, 0.2814887762069702, 0.8350613117218018, 0.1543942242860794, -1.241218090057373, 0.12360338866710663, -0.8461910486221313, -1.0086758136749268, 0.3181535601615906, 0.7738444805145264, 0.25155842304229736, 0.15914765000343323, -1.00927734375, -0.9864212870597839, 0.5110775828361511, -0.15278032422065735, -0.17036977410316467, -0.5273115038871765, -0.5751434564590454, 0.44904667139053345, 0.16830162703990936, 0.2011709064245224, -1.6198620796203613, -1.2563509941101074, 0.8502961993217468, -0.3263196647167206, 0.030390717089176178, -0.45399582386016846, -0.5866563320159912, 0.6661009192466736, 0.16955125331878662, -0.011661720462143421, 0.3290654420852661, 2.452981472015381, 2.3418238162994385, 0.2479112446308136, 1.2282195091247559, 0.19030913710594177, 0.1867225170135498, 0.3293505012989044, -0.07487862557172775, -0.4470890462398529, 0.6928691267967224, 0.4573929011821747, 0.2558267414569855, -1.2584375143051147], [-2.5127735137939453, 0.4340862035751343, 1.34297513961792, -0.3756662905216217, -0.7472779750823975, 0.5241796374320984, 0.6614407300949097, -0.1359390914440155, 0.7698924541473389, -0.12290407717227936, 0.5634666085243225, 0.42467954754829407, 0.10959052294492722, -0.8348851799964905, 0.5534823536872864, -0.5865735411643982, -1.3188073635101318, 0.4108748137950897, 0.6590614318847656, -0.5213049054145813, 0.6465794444084167, 0.07807574421167374, 0.8164312243461609, -0.2577945291996002, 0.18302901089191437, -0.3704979419708252, 1.3695292472839355, -0.5201085805892944, 1.4727751016616821, 1.3233970403671265, 0.07290937006473541, 1.2527559995651245, 1.13774836063385, 1.2395853996276855, 0.4695228040218353, -1.5785207748413086, -1.099686622619629, 0.21202081441879272, -0.09512187540531158, -0.23574592173099518, -0.27746978402137756, 1.3123654127120972, -0.04186318814754486, 1.6269574165344238, -0.672453761100769, -0.6399596333503723, 0.5601249933242798, 2.1503794193267822, 0.39160847663879395, -1.9430644512176514, 0.41298454999923706, -0.9748726487159729, -0.3597507178783417, -1.0805931091308594, 0.6913993954658508, -0.6173802614212036, 1.236655592918396, 0.5629696846008301, 0.09002473205327988, -0.5036361217498779, -1.6203596591949463, 0.48641419410705566, 0.4475467801094055, 2.022580146789551], [-0.712429404258728, 0.6693311929702759, 0.403507798910141, 1.2023729085922241, -0.08878761529922485, -0.9224030375480652, -0.307131290435791, -0.6850391030311584, -0.8974555730819702, -1.3006761074066162, 0.03183948993682861, -0.030369609594345093, -0.08845870941877365, -2.4531681537628174, -1.5744404792785645, 0.3662067651748657, -0.5640270709991455, -0.10837070643901825, -0.24958163499832153, 0.3092298209667206, -1.7158544063568115, 1.403169870376587, 0.5784608125686646, -1.5183361768722534, -1.344379186630249, 0.6153753995895386, -0.3704473078250885, -2.0435221195220947, 0.39071282744407654, 0.038031820207834244, -0.2478901892900467, 0.13001565635204315, 1.2976133823394775, -1.2203781604766846, -1.2677584886550903, 0.263244092464447, 0.05103062465786934, 1.7158069610595703, 0.5430864691734314, 0.8293555974960327, 2.4179155826568604, -1.244891881942749, -0.18774203956127167, -0.5712841749191284, 0.7071697115898132, 1.1605408191680908, -1.1529771089553833, -0.4916784465312958, 0.4451877772808075, 0.27894464135169983, -0.16757243871688843, 0.1277429312467575, 0.0890326276421547, 1.0975642204284668, -0.7340695858001709, -0.44891080260276794, 0.951208770275116, -0.05170051008462906, 0.18594984710216522, -1.4724596738815308, -0.3912123739719391, -0.1438402533531189, -1.2355955839157104, 0.14483238756656647], [-2.350029945373535, -0.5013860464096069, -0.5871612429618835, -0.3337731957435608, 0.6752519607543945, 0.7129024267196655, -1.1418776512145996, 0.044697776436805725, 3.06317400932312, -0.7123656868934631, 0.9284748435020447, 1.982434630393982, 1.6143625974655151, 1.8246620893478394, -0.48149335384368896, -0.9566383957862854, 2.2453150749206543, -1.400975227355957, 0.9966917634010315, -0.3908098042011261, -0.6152279376983643, -0.07643581181764603, 0.6401268839836121, 1.6577473878860474, -0.16413000226020813, 1.4504386186599731, 0.618240237236023, -0.14393173158168793, 0.7669983506202698, -1.1936416625976562, 1.25091552734375, -0.40922805666923523, 0.48093563318252563, 1.2050851583480835, 0.31292399764060974, 0.3780800402164459, 0.22214867174625397, 0.6396923065185547, 0.03298193961381912, 2.3312952518463135, 0.30572593212127686, 0.05433082953095436, -0.8876454830169678, 0.6750696897506714, 0.15651631355285645, 0.0972544401884079, 1.7233572006225586, 0.3586190640926361, -0.5345441102981567, -1.5961195230484009, 0.3288823068141937, -0.5563465356826782, 0.3897830843925476, -1.040314793586731, 0.3611922264099121, 1.469307780265808, -0.5775585770606995, -1.0785447359085083, 0.4244687557220459, -1.0233367681503296, 0.4332050085067749, 1.0935020446777344, -0.7709248065948486, 1.2653625011444092], [0.5002008080482483, -0.08768349885940552, -0.36288318037986755, -0.7765483856201172, -0.33961978554725647, 0.8320597410202026, -0.22887541353702545, -0.11704997718334198, 0.7409077286720276, -0.28959694504737854, -0.7102479338645935, -1.7113784551620483, -1.676431655883789, 0.8409538865089417, 1.8356335163116455, 1.5993057489395142, -1.2423372268676758, 0.05809951201081276, 0.1495625227689743, -2.536125421524048, -1.3805803060531616, -0.10491635650396347, 0.04701891541481018, 2.018709659576416, -0.5632755756378174, 1.9460965394973755, 1.2865628004074097, 0.1867372691631317, 0.2642667591571808, 0.35422250628471375, 2.3826780319213867, 0.628088653087616, 0.755385160446167, 0.11145545542240143, -0.819564163684845, -0.23996026813983917, 0.5300528407096863, 0.9966612458229065, 0.3411109149456024, -0.30775195360183716, 0.6054113507270813, -1.5765774250030518, 1.572496771812439, -0.6093950867652893, 0.4676651656627655, 1.183321475982666, 0.2505338788032532, -0.8104475140571594, 0.3609448969364166, 2.351215362548828, -0.2210337072610855, -0.3532325029373169, -0.061082277446985245, -0.5396047234535217, -2.0725607872009277, 0.2097117155790329, -0.4505864679813385, 0.7397327423095703, 0.06819469481706619, -0.6942784786224365, 0.8442889451980591, -1.0132651329040527, 0.16701963543891907, 1.8621151447296143], [0.31218862533569336, 1.0473358631134033, 0.14039915800094604, 0.2798755168914795, 0.0822010412812233, -1.775494933128357, 1.0939271450042725, -0.4518146216869354, -0.06478577107191086, 0.8430626392364502, 1.0266497135162354, 3.7387847900390625, 0.10586114227771759, -0.19801568984985352, 1.711151361465454, -0.9994291663169861, 0.905176043510437, 0.6836636662483215, 0.9956881999969482, 2.7090904712677, 0.7409300804138184, -0.5184612274169922, -2.1014060974121094, -0.12795577943325043, -0.6107354760169983, 0.5897424221038818, 2.08111572265625, -3.0405337810516357, 0.6342077851295471, 1.2726982831954956, -1.408546805381775, -1.9998531341552734, -0.3471202850341797, -0.7956160306930542, -0.16087780892848969, 0.8307160139083862, -1.928546667098999, -0.5832752585411072, -0.04796953871846199, -0.745245099067688, 1.9378694295883179, 1.1867941617965698, -0.748258650302887, 1.8464418649673462, -1.123309850692749, 0.29049772024154663, -1.2479031085968018, -2.6278316974639893, -0.269536554813385, -0.2903437912464142, -0.33192917704582214, -0.2266417294740677, 0.7952477931976318, -1.0169883966445923, 0.32993313670158386, 1.0299077033996582, 1.797086238861084, -0.02447199448943138, -0.44179999828338623, -0.8063907027244568, 1.046298861503601, -0.0009672902524471283, 1.6213507652282715, -0.9774649739265442], [-0.6700806617736816, -0.750382125377655, 2.6301870346069336, 0.5092610716819763, 1.015453815460205, 0.23810185492038727, 0.3109455704689026, -0.6657076478004456, -1.260504961013794, 0.1313316524028778, 0.019357284530997276, -0.2978533208370209, 0.6036463379859924, -0.15647678077220917, -0.4095749855041504, 0.4167191982269287, 0.741306483745575, 1.3915841579437256, 0.3331354260444641, -0.318416953086853, -0.2195032387971878, 0.80381840467453, -0.07602176815271378, -0.07339385151863098, -0.5802620053291321, 0.9320034384727478, 0.6488269567489624, -1.0318183898925781, 0.8319658041000366, -0.3507164418697357, 1.3349223136901855, -0.25230419635772705, -2.4390766620635986, 2.4856529235839844, -1.099297285079956, -0.9097693562507629, -0.9006792306900024, 1.7636412382125854, 0.3270833194255829, 0.3424909710884094, 0.6515206694602966, -0.2751859724521637, -0.13415683805942535, -0.6140331625938416, 0.30790188908576965, 0.5137651562690735, 0.018090832978487015, 1.7646479606628418, 0.2910686433315277, -0.29307374358177185, 0.12530456483364105, -0.48315057158470154, 0.3677882254123688, -0.7791600823402405, -0.836350679397583, -0.02338731847703457, -1.1753993034362793, 0.4595094323158264, 0.5414132475852966, 0.15229864418506622, -0.8485085368156433, -0.7271339297294617, 2.052762031555176, -1.1202877759933472]], "dtype": "float32"}, 8, 64], "e": {"__tup__": [{"__nd__": [[-21.74650001525879, -62.30982208251953, 2.8431928157806396, 26.172605514526367, -28.645227432250977, 68.302734375, -0.8315674066543579, -58.77351379394531, -42.505096435546875, -51.37416076660156, 47.706092834472656, -38.52497863769531, 5.506323337554932, -41.84793472290039, -63.599403381347656, 11.912968635559082, 22.026386260986328, 74.85333251953125, 24.223587036132812, -2.628235101699829, 27.356382369995117, -78.9732437133789, 52.34468460083008, 6.111263751983643, -9.478313446044922, 87.09306335449219, 41.586490631103516, -22.924348831176758, 53.34359359741211, 111.5769271850586, 36.136680603027344, 22.379159927368164, 25.77629280090332, 16.76190948486328, -1.2410939931869507, 2.443455457687378, 55.559391021728516, -54.42119216918945, 48.95956802368164, 20.950082778930664, -21.0421199798584, 56.31357955932617, 60.969730377197266, -37.824710845947266, 48.756103515625, -42.3891487121582, 8.858573913574219, -58.74993896484375, -45.279117584228516, -21.73567008972168, -3.8147342205047607, -41.608360290527344, -0.3979279696941376, 11.520271301269531, -63.01445388793945, 16.160837173461914, -0.5629121661186218, -94.62242126464844, 42.8809814453125, 11.916172981262207, -77.15005493164062, -63.47361373901367, 9.582841873168945, 21.0848445892334], [19.084749221801758, 26.454702377319336, 39.47294235229492, -100.66712951660156, -12.726404190063477, -13.794201850891113, 8.458224296569824, 64.58391571044922, -108.19760131835938, 56.29802322387695, -81.9990005493164, -1.4798524379730225, 0.13716508448123932, -37.0447998046875, 10.571300506591797, -50.2015266418457, -24.6998291015625, -36.519371032714844, -11.12354850769043, -8.79251766204834, 6.374946117401123, -47.98480224609375, -38.223785400390625, -42.594078063964844, -11.27027416229248, 19.77977752685547, 6.165102005004883, -17.093917846679688, -31.321239471435547, 19.11721420288086, -29.22972869873047, 6.9221696853637695, -34.810325622558594, 14.821351051330566, -106.59761810302734, 8.149579048156738, 13.712934494018555, 32.54271697998047, 10.582320213317871, -15.196836471557617, -78.41432189941406, 25.94002914428711, 91.65078735351562, 16.534393310546875, 64.21695709228516, 6.687480449676514, -13.048306465148926, 29.317399978637695, 39.56919479370117, -50.41304016113281, -24.463895797729492, 13.102487564086914, 28.007694244384766, -24.958576202392578, 72.68315887451172, 22.859638214111328, -21.855880737304688, -15.624917984008789, 21.647706985473633, 33.483123779296875, 35.812225341796875, 6.800158977508545, 6.216615676879883, -53.481590270996094], [-36.43728256225586, 93.62576293945312, 85.82640075683594, -105.70963287353516, -4.740242958068848, 11.004043579101562, 2.4592783451080322, 99.09894561767578, 2.807920455932617, 63.16778564453125, 32.966983795166016, 3.5709192752838135, -8.585546493530273, -69.40160369873047, 8.029847145080566, 91.37073516845703, -142.0636444091797, 75.81143951416016, 50.70409393310547, 6.768120765686035, 14.64132022857666, -14.058162689208984, -106.96269989013672, 144.6464385986328, 30.89012336730957, -51.25176239013672, -4.825267314910889, 87.35337829589844, -5.324047565460205, -11.08492660522461, 38.26533889770508, -36.722251892089844, 45.8076171875, 58.39561080932617, -16.064144134521484, 22.10849380493164, -7.358062744140625, 32.02760314941406, 61.54169464111328, -6.992757320404053, 56.8336296081543, 47.32328414916992, 31.234294891357422, 74.4347915649414, -5.676385879516602, -11.171320915222168, -38.147064208984375, 6.517638683319092, 33.94722366333008, -5.933914661407471, -11.438416481018066, -18.780914306640625, -114.24222564697266, 107.48654174804688, 28.369665145874023, -15.853227615356445, 62.28337097167969, 20.102033615112305, -4.940729141235352, 15.795002937316895, 17.961767196655273, 56.52605056762695, 70.48963928222656, -1.206720232963562], [39.566009521484375, -126.03312683105469, -18.626155853271484, -42.90355682373047, 18.660076141357422, 19.587738037109375, -5.089913845062256, -41.664085388183594, 13.882872581481934, -30.762659072875977, 26.261920928955078, 16.27254295349121, 65.41455841064453, 2.003523826599121, -1.520736575126648, 14.020525932312012, 16.483001708984375, 60.972740173339844, 16.1061954498291, 13.07312297821045, -10.758014678955078, -55.39189529418945, 16.09056854248047, 34.90903854370117, 34.0577507019043, 12.053899765014648, 115.46077728271484, 44.5457878112793, 20.981098175048828, -24.32862663269043, 47.50583267211914, 50.52623748779297, -103.12033081054688, -57.177207946777344, -27.408973693847656, 18.781530380249023, 26.86163902282715, 10.171466827392578, -23.654645919799805, -8.03671646118164, 23.757455825805664, 43.346160888671875, 58.50651550292969, -20.088085174560547, 73.93524932861328, -75.80758666992188, -31.52452278137207, -33.25380325317383, -22.174678802490234, -55.714935302734375, 96.47738647460938, 25.85979461669922, 37.61897277832031, -33.72677230834961, 41.97855758666992, -38.52351379394531, 4.650768280029297, 17.58929443359375, -12.201333999633789, 73.7221450805664, 11.227910995483398, -27.493579864501953, -6.517559051513672, 26.16119384765625], [-36.379188537597656, -7.961600303649902, 15.16870403289795, -37.646820068359375, -44.93903350830078, 0.8398889899253845, 30.360591888427734, -66.138916015625, -35.841270446777344, -27.244922637939453, 36.756263732910156, -6.344505310058594, 51.18791198730469, 27.66750717163086, -15.219715118408203, 58.49542999267578, 73.64253997802734, -23.015628814697266, -1.7180297374725342, 59.9869499206543, 56.640594482421875, -47.96354293823242, 138.50381469726562, -80.29483032226562, 6.997081756591797, -18.292404174804688, 32.16480255126953, 29.384916305541992, 55.550045013427734, -29.404693603515625, -19.16671371459961, -18.264118194580078, 5.1416802406311035, 107.3556137084961, -35.400733947753906, 33.03505325317383, -50.247310638427734, -81.3176498413086, -61.11912536621094, 29.996583938598633, -61.03168869018555, 43.6462287902832, 46.56964111328125, -0.5044155716896057, -25.14932632446289, 23.106586456298828, 1.7863339185714722, 38.00554656982422, 57.7498779296875, 50.54390335083008, 0.771501898765564, -102.57867431640625, -59.5081672668457, 31.68830680847168, -98.20646667480469, 57.6763916015625, 0.7010387778282166, 31.979942321777344, 47.0059928894043, -26.94515037536621, 52.53101348876953, 11.658920288085938, -0.95834881067276, 21.893474578857422], [-116.10677337646484, -50.4749870300293, -38.57659912109375, -29.48682975769043, -19.902103424072266, -148.71922302246094, 18.82600975036621, -43.46648406982422, 133.56141662597656, 67.16846466064453, -9.452834129333496, 2.539663314819336, -49.134098052978516, -4.912109851837158, -76.58123779296875, -53.73466873168945, 18.87816619873047, 21.86883544921875, 24.237457275390625, -36.812294006347656, -31.39260482788086, 77.09051513671875, -97.51286315917969, -41.15178680419922, 59.65923309326172, -50.95485305786133, 7.590889930725098, 76.77175903320312, 46.205047607421875, 108.8187026977539, -19.699966430664062, -20.416303634643555, 43.22225570678711, 56.66752243041992, -125.24368286132812, -59.1849250793457, 13.298873901367188, -27.375213623046875, -57.7582893371582, 29.970483779907227, 26.899913787841797, 4.474655628204346, 18.685672760009766, 35.93370819091797, -4.625005722045898, 5.066359996795654, -39.59817886352539, -41.40209197998047, -31.346866607666016, -61.75993347167969, -30.024555206298828, -36.47236633300781, -27.31861114501953, -7.5480804443359375, 55.491966247558594, 68.9959487915039, -2.858628511428833, 10.55251693725586, -50.29268264770508, 42.39958190917969, 32.236297607421875, 135.21282958984375, -26.63987922668457, -43.530818939208984], [44.74055099487305, -63.1943473815918, -14.302955627441406, -37.045658111572266, 4.859391212463379, -70.33509063720703, 10.980284690856934, -109.02690887451172, 3.828174591064453, 71.35075378417969, -43.5526123046875, 50.46731948852539, 5.52079439163208, -6.5650811195373535, 1.3256670236587524, -105.54985809326172, 3.4532055854797363, -29.526260375976562, -77.09242248535156, 9.826461791992188, 14.8901948928833, 67.31727600097656, -37.76252365112305, 20.19753074645996, -29.810197830200195, -47.434696197509766, 100.26268768310547, -77.30332946777344, -22.114837646484375, 11.53007698059082, 64.32444763183594, 93.75943756103516, 19.52704620361328, -41.40500259399414, 28.55370330810547, 53.78166198730469, 63.85906219482422, 16.09241485595703, -37.06847381591797, -8.985647201538086, -13.183635711669922, 8.944685935974121, 22.3228702545166, -33.328304290771484, 60.106624603271484, -66.26891326904297, 33.739585876464844, 37.17067337036133, 105.0434799194336, -14.87434196472168, 96.17823791503906, 74.83924102783203, -64.30670166015625, -36.148712158203125, -8.3956937789917, -22.202251434326172, -71.64236450195312, 40.1309814453125, 45.264801025390625, 73.08065795898438, -28.027050018310547, -40.23431396484375, -46.26032257080078, 19.76808738708496], [-30.472200393676758, -42.542015075683594, -10.808815002441406, 71.46154022216797, 9.463343620300293, -34.9085807800293, 29.252225875854492, 15.059188842773438, -6.602267742156982, -81.04107666015625, -54.77934646606445, 37.256351470947266, 78.33905792236328, -21.513553619384766, -61.144412994384766, 1.8435560464859009, 15.70859432220459, -83.44188690185547, -15.047060012817383, 26.683359146118164, 39.55780792236328, 71.87872314453125, 11.949865341186523, 14.518450736999512, 87.93209075927734, -35.34329605102539, 15.152878761291504, 7.338959217071533, -33.423587799072266, 33.06705856323242, 28.16625213623047, 40.509246826171875, -53.55881118774414, -22.18140411376953, -23.08172607421875, 17.673044204711914, 98.89213562011719, -5.831568717956543, -33.841949462890625, -46.78968048095703, -15.482101440429688, 57.107723236083984, 88.65998077392578, 28.581296920776367, -3.1169540882110596, -76.4501724243164, 82.90476989746094, -30.695453643798828, -49.81074142456055, -66.18810272216797, -24.440811157226562, -35.33121109008789, 30.479955673217773, 33.326358795166016, -73.823974609375, -29.26211166381836, -3.718496322631836, 36.36833572387695, 38.906314849853516, -29.317855834960938, -43.69729995727539, -53.706031799316406, 177.4893341064453, 79.36919403076172], [24.24995994567871, -79.7262191772461, 26.076478958129883, 20.717615127563477, 29.10433578491211, -13.19626522064209, 14.995936393737793, -53.476417541503906, 47.94957733154297, 6.64829158782959, -59.79616165161133, 46.63091278076172, 20.339284896850586, -43.34251022338867, 29.909034729003906, 19.5504207611084, -93.79640197753906, 44.54063415527344, 52.538448333740234, -13.727997779846191, -45.76420211791992, 86.46453094482422, -26.065776824951172, -48.09244918823242, 84.67127990722656, -9.062993049621582, 49.091495513916016, 32.600830078125, -59.16822814941406, -87.13275909423828, -26.67644691467285, -42.87339782714844, -3.2827680110931396, -27.137039184570312, -58.654842376708984, -66.4939193725586, -32.15456771850586, 54.894168853759766, 94.75184631347656, 13.637079238891602, -8.33272647857666, -35.73195266723633, 48.85799026489258, -73.62232971191406, 20.665922164916992, 43.72977828979492, 11.827459335327148, -8.611451148986816, -62.1104850769043, -25.47948455810547, 50.46644973754883, 14.67686939239502, -7.824517726898193, -77.90504455566406, -56.14909744262695, 25.68923568725586, -20.307926177978516, 21.98530387878418, -56.28144073486328, 10.144001960754395, -0.856070339679718, -13.137974739074707, -5.979031562805176, 29.24873161315918], [-31.552244186401367, -13.02398681640625, 29.108190536499023, 62.275150299072266, 33.556549072265625, -58.784828186035156, -11.42296314239502, 83.7079086303711, -55.118621826171875, 19.619966506958008, -52.098731994628906, 23.38356590270996, 20.123958587646484, -14.546625137329102, 3.366554021835327, -8.957423210144043, 39.75876235961914, -76.50397491455078, 27.521236419677734, 44.667869567871094, 99.676025390625, -55.11532211303711, 23.268102645874023, -2.1701765060424805, 1.112046241760254, -48.7822265625, 15.266191482543945, -74.23607635498047, -87.27079772949219, 89.17414093017578, -22.3684139251709, -34.13021469116211, 27.946218490600586, 8.775997161865234, -24.70376968383789, -34.1035270690918, 24.621307373046875, 25.569198608398438, -4.86285924911499, 19.691762924194336, 25.753889083862305, 21.923791885375977, -58.357608795166016, 56.6737060546875, 19.905155181884766, 43.94102478027344, 49.999881744384766, -17.994728088378906, 40.81718826293945, -44.77505874633789, -70.47486877441406, -27.646465301513672, 1.7847232818603516, -37.3589973449707, 66.44718933105469, -2.7587742805480957, 94.8678970336914, 45.009925842285156, 74.46768951416016, -14.256660461425781, 44.82624435424805, -15.841143608093262, -6.448707103729248, -37.84723663330078], [65.63444519042969, 12.006694793701172, 36.77413558959961, 47.187198638916016, 10.781400680541992, -1.4598908424377441, 68.91671752929688, 27.97834587097168, 0.18076974153518677, 27.532873153686523, 32.022647857666016, 33.76118850708008, -10.987032890319824, 44.12073516845703, 85.20775604248047, 10.099447250366211, -73.940185546875, 58.64851760864258, 35.59550476074219, 21.12381935119629, 118.47184753417969, -47.4693603515625, -50.720924377441406, -3.9926834106445312, 35.30135726928711, -3.8744115829467773, 20.25965118408203, -68.09262084960938, -34.5259895324707, 33.92829895019531, -101.990966796875, 65.69051361083984, 44.55008316040039, 0.5135777592658997, -33.71368408203125, -23.595111846923828, 12.157576560974121, -39.866554260253906, 1.7106455564498901, -10.909461975097656, -12.405896186828613, 13.36410903930664, 20.64575958251953, 29.63848304748535, 35.051876068115234, 47.25213623046875, 11.373422622680664, 19.140581130981445, 16.071657180786133, -76.54964447021484, -15.37548542022705, 14.967575073242188, 38.977718353271484, 54.12858963012695, -3.2820262908935547, 5.075328826904297, 72.56320190429688, -81.24687957763672, 37.047325134277344, -50.84318542480469, -9.307046890258789, -19.948627471923828, 81.53924560546875, 29.558914184570312], [12.516627311706543, 57.22974395751953, 25.52851104736328, -19.36908721923828, 94.0441665649414, 11.20032024383545, -43.64043426513672, 30.969797134399414, 53.6918830871582, -2.6087679862976074, -96.30548095703125, 14.591712951660156, 2.436818838119507, -32.413352966308594, 119.43800354003906, 23.2214412689209, -42.207008361816406, 20.22454833984375, -1.6066737174987793, -18.132726669311523, 26.848663330078125, -98.69702911376953, 16.500314712524414, -56.03956604003906, 22.007862091064453, -45.428592681884766, 72.06536865234375, 44.539737701416016, 96.67445373535156, -27.905431747436523, -40.416011810302734, -47.740867614746094, -5.180006504058838, 31.855833053588867, 11.031707763671875, 80.48504638671875, 58.16844177246094, -54.14745330810547, 14.92483901977539, 10.192010879516602, 60.38185119628906, 87.38602447509766, 13.68366813659668, 67.02889251708984, -89.21145629882812, 87.15319061279297, 101.26121520996094, 49.64657211303711, -10.071039199829102, -39.9795036315918, 6.273606300354004, -35.66925811767578, -92.82432556152344, -89.08726501464844, 28.700531005859375, -98.3858871459961, 20.530736923217773, -124.38653564453125, -45.04393005371094, -86.47826385498047, 39.70335388183594, -40.6037483215332, 13.20569896697998, 64.52767181396484], [1.6642248630523682, -77.31739807128906, -13.222208023071289, 26.56370735168457, -11.290523529052734, 30.662967681884766, 75.18321228027344, -7.497674942016602, -1.913658618927002, 14.761821746826172, -21.756784439086914, 2.5387508869171143, 28.730615615844727, 9.210742950439453, 31.459197998046875, 14.971338272094727, -48.819210052490234, -10.246566772460938, 62.051246643066406, 0.24349237978458405, 62.33387756347656, -57.66309356689453, -64.42670440673828, -115.73223114013672, 60.118125915527344, 45.24442672729492, 39.48820114135742, -36.72187805175781, -10.448356628417969, 46.83444595336914, 13.383466720581055, 106.22949981689453, -40.18770217895508, -11.17978572845459, 2.1184682846069336, 10.121971130371094, 60.808475494384766, 115.58169555664062, 17.614198684692383, 4.138917922973633, 24.529525756835938, -60.1174201965332, -5.62472677230835, 8.343204498291016, -107.0202407836914, 42.76498794555664, 2.879643440246582, 13.613411903381348, 39.11595916748047, 55.034423828125, 54.47372055053711, -28.981172561645508, 18.98968505859375, -3.3739378452301025, -39.094356536865234, -81.82129669189453, -12.249933242797852, 51.691139221191406, -75.48200225830078, -73.4982681274414, -75.89762115478516, 45.800392150878906, -71.00166320800781, -59.99687194824219], [20.047433853149414, 83.93797302246094, -60.41865921020508, -44.42494583129883, 30.596651077270508, -96.87007141113281, 36.07318115234375, -20.869049072265625, -14.039636611938477, -79.19180297851562, -44.53870391845703, -29.996957778930664, 16.124610900878906, -28.810148239135742, 37.98121643066406, 89.59342956542969, 52.58189010620117, -58.067543029785156, -37.023536682128906, 31.37125587463379, 7.440921783447266, -10.827750205993652, -66.6767349243164, -33.79367446899414, 21.479299545288086, 38.518680572509766, 73.62776947021484, -36.09168243408203, -6.683814525604248, 66.58057403564453, -162.03964233398438, -57.14105987548828, 44.93924331665039, 8.136688232421875, 26.66696548461914, -20.136096954345703, -17.065975189208984, 35.87103271484375, 20.160202026367188, -21.097156524658203, -46.13289260864258, 4.145566940307617, -85.59151458740234, -16.338882446289062, -19.43339729309082, -30.64651870727539, 8.032779693603516, -88.74393463134766, 46.272216796875, 6.140554428100586, -61.86784744262695, -18.924083709716797, -6.8075385093688965, 9.932295799255371, 45.247440338134766, -14.006890296936035, 39.64240264892578, -22.476152420043945, 95.04963684082031, 61.4413948059082, 17.768556594848633, 37.8262939453125, -15.606485366821289, 4.971614837646484], [-71.87439727783203, -44.66234588623047, -28.23348045349121, -20.661235809326172, -62.952125549316406, 71.88764953613281, 28.32734489440918, 123.76013946533203, 10.16823959350586, 7.194646835327148, -15.983338356018066, -26.455432891845703, 3.297882080078125, -15.719205856323242, -35.98793411254883, -94.67790222167969, 45.406524658203125, -24.501728057861328, 50.22203063964844, 66.84398651123047, -5.186424732208252, 80.352294921875, -46.72956085205078, -26.619651794433594, -21.035158157348633, 14.941971778869629, -41.852046966552734, 5.701984405517578, 39.57291030883789, -23.031545639038086, 15.66026782989502, -69.91230773925781, -85.04930114746094, 56.2341194152832, -24.170642852783203, 91.4434814453125, 107.98844909667969, 75.3060302734375, 57.45873260498047, -68.6587905883789, -79.69617462158203, 29.15618896484375, 0.3572181165218353, -59.67155075073242, 60.5439338684082, 17.036605834960938, -41.815494537353516, -14.180550575256348, -43.37397766113281, 12.54090690612793, -34.18928527832031, -65.76787567138672, -17.499923706054688, 14.609782218933105, -3.8829805850982666, 56.94818115234375, -112.02404022216797, 13.49938678741455, -76.534423828125, 80.55665588378906, 46.51371383666992, 23.834251403808594, -77.85446166992188, -16.940832138061523], [-40.18770217895508, -10.808850288391113, 5.219825744628906, -36.70035171508789, 11.731782913208008, -41.573673248291016, 19.73467445373535, -9.977408409118652, -45.53244400024414, -27.017990112304688, -85.80809783935547, 11.263635635375977, 42.76348876953125, 24.257009506225586, -83.11483001708984, 29.685853958129883, 42.704750061035156, -36.72597885131836, -29.860227584838867, -63.490264892578125, 49.51171112060547, 70.7579345703125, -11.347070693969727, 54.4776725769043, 5.411362648010254, 12.829524993896484, -8.816137313842773, 134.37461853027344, 5.059666633605957, 25.521692276000977, -8.732250213623047, 6.14573335647583, 69.41679382324219, -61.972190856933594, -18.191762924194336, -51.795780181884766, -2.1668431758880615, -34.71658706665039, 18.871868133544922, 55.652957916259766, 20.07847023010254, 53.08045196533203, 28.913917541503906, -1.1604983806610107, -52.4408073425293, -57.770851135253906, 69.16012573242188, 29.16298484802246, -31.846229553222656, -16.08634376525879, -50.62382125854492, -13.820719718933105, 4.278505325317383, 31.015295028686523, 15.372397422790527, 9.81774616241455, 44.88492965698242, -26.534841537475586, 9.851005554199219, -64.79307556152344, 44.38602066040039, 21.761409759521484, 57.80793762207031, 44.55490493774414]], "dtype": "float32"}, {"__nd__": [[[-12.913386344909668, 9.195965766906738, 20.19923210144043, 9.615374565124512, 20.90431022644043, 13.595332145690918, -0.6803363561630249, 1.659804344177246, 16.870134353637695, -25.221351623535156, 14.833325386047363, -1.6247353553771973, 29.002912521362305, 7.728693962097168, -10.54975700378418, 0.29215148091316223, 17.528371810913086, 7.9492974281311035, -6.077855110168457, 12.564905166625977, 4.978814125061035, -15.455060005187988, -8.011393547058105, -17.493736267089844, 9.928601264953613, 16.595041275024414, -13.323554039001465, -8.059155464172363, -5.437008380889893, 23.51371955871582, 8.28249454498291, -10.559309005737305, 5.60255241394043, 8.31204605102539, 27.116310119628906, -20.416563034057617, 5.8247761726379395, 13.034838676452637, 11.012269020080566, 3.606365919113159, 11.444951057434082, 42.15515899658203, 33.85888671875, 16.427003860473633, -1.6442149877548218, -14.479023933410645, -7.651744365692139, -20.363155364990234, -13.094545364379883, -0.7456613779067993, -6.781497955322266, -28.640453338623047, -14.077247619628906, -4.861978530883789, -6.203662872314453, 25.520614624023438, 23.368419647216797, -42.33138656616211, -4.0367112159729, -17.12897491455078, 11.716780662536621, -9.820832252502441, 9.141880989074707, 17.01660919189453], [4.162899494171143, 16.420673370361328, 28.280479431152344, 2.5165889263153076, -30.43477439880371, -19.24282455444336, -2.2724359035491943, 11.108589172363281, -0.8482039570808411, 3.276089906692505, -14.04495906829834, 20.337980270385742, 4.710020065307617, 13.10486888885498, 10.307232856750488, 2.279465913772583, -16.112648010253906, 2.791750192642212, 19.115528106689453, 2.031946897506714, -5.1620683670043945, -16.855003356933594, 11.934020042419434, -19.395029067993164, -20.930540084838867, 40.85612487792969, -4.86395788192749, 25.199007034301758, -12.843218803405762, -9.957171440124512, 22.71155548095703, -3.4914565086364746, -4.699370384216309, 7.651291847229004, -18.179542541503906, -3.7835512161254883, 1.652876853942871, 3.36582612991333, 9.597949981689453, 24.218923568725586, 15.404186248779297, -27.802410125732422, 19.995437622070312, -0.07608740776777267, -4.601916790008545, 4.889184951782227, -20.46246910095215, 18.072853088378906, -16.429224014282227, -10.45091438293457, 1.1995739936828613, 13.940648078918457, 10.456108093261719, -13.770805358886719, 3.2846312522888184, -10.763751983642578, 5.897421360015869, -18.735654830932617, -5.69165563583374, 3.166131019592285, 10.9028902053833, 12.687029838562012, -3.4974915981292725, -6.600608825683594], [19.82073211669922, 2.4140357971191406, 10.809067726135254, -4.551313400268555, 23.390043258666992, 15.78093147277832, 6.125931739807129, 24.02557373046875, 27.50908660888672, -2.4242732524871826, 2.2358946800231934, 54.15476989746094, -5.070599555969238, -7.078375816345215, -37.665889739990234, -27.651081085205078, -10.916529655456543, 6.202817440032959, 9.025251388549805, -17.389999389648438, 10.313922882080078, 22.968503952026367, -26.79620933532715, 26.58127212524414, -31.380895614624023, -0.6895705461502075, -15.281173706054688, 3.6596248149871826, -15.656560897827148, 30.997879028320312, -11.385452270507812, -2.2652106285095215, 23.086977005004883, 19.27805519104004, 13.87652587890625, 7.41802453994751, 5.978904724121094, 9.533806800842285, 21.9835147857666, 16.349117279052734, 0.9332942366600037, -8.992851257324219, -27.555391311645508, 9.884736061096191, 21.132339477539062, 7.78715705871582, 11.855835914611816, -15.601438522338867, 0.6366744637489319, 10.107810974121094, -14.845107078552246, -1.0558335781097412, -20.401338577270508, 12.675616264343262, -9.872271537780762, -11.840289115905762, 9.927576065063477, -2.9687886238098145, -14.92293643951416, 3.8569700717926025, -2.0253005027770996, 7.642632484436035, -29.309181213378906, 10.552335739135742], [10.12978744506836, -7.800473690032959, 1.0681089162826538, -17.118118286132812, 4.1375322341918945, 17.893505096435547, -4.824820041656494, 17.234697341918945, 6.206362724304199, -15.92595386505127, 15.864812850952148, -22.708982467651367, 26.99692153930664, 10.95373249053955, -14.702534675598145, 5.968935966491699, 24.32792854309082, -6.450443267822266, 17.43052864074707, -24.269575119018555, -5.404823303222656, 3.3425590991973877, 3.3959767818450928, -1.702834963798523, 17.339937210083008, 20.979162216186523, 8.739197731018066, 12.80300235748291, 15.514998435974121, -2.64788556098938, 10.806382179260254, 31.978443145751953, -13.3637056350708, -15.002004623413086, -34.98486328125, -1.8769729137420654, -23.13808822631836, 0.16701751947402954, -34.669151306152344, -15.609840393066406, 14.410623550415039, -2.587179660797119, 12.71159553527832, -4.174768447875977, -9.604447364807129, -32.08897399902344, -2.9784321784973145, -18.680118560791016, -14.102828025817871, -2.2116103172302246, 6.8577094078063965, 15.724650382995605, -13.23617172241211, -30.757104873657227, -5.960236549377441, -26.328460693359375, 12.832197189331055, 14.858078956604004, -10.543630599975586, 2.072385549545288, -6.6749043464660645, 4.1325483322143555, -25.759061813354492, -10.057291984558105], [-3.55869197845459, 2.3749475479125977, 9.936403274536133, 3.1677446365356445, -9.162264823913574, 23.113262176513672, 1.2822679281234741, 8.352897644042969, -9.354025840759277, -2.196718454360962, 28.0885009765625, 12.130206108093262, 24.980741500854492, 20.971879959106445, 5.4013824462890625, 14.090177536010742, -0.5228756666183472, 4.975048065185547, 31.984865188598633, -10.410447120666504, 13.738731384277344, 21.21935272216797, 9.301451683044434, 5.504175662994385, -8.738454818725586, -10.485133171081543, -11.033941268920898, 12.405893325805664, -27.240169525146484, 7.7444658279418945, 4.505246639251709, -26.910634994506836, 2.7175164222717285, 10.419529914855957, -1.7085462808609009, -4.983523368835449, 12.916818618774414, -22.70969009399414, 3.028184175491333, -6.761488914489746, 14.858939170837402, 8.036301612854004, 7.657148838043213, 2.2945098876953125, 13.227210998535156, -15.66290283203125, -3.105778694152832, -1.1714832782745361, 3.9427971839904785, 29.915409088134766, 27.454832077026367, -17.592039108276367, 12.879249572753906, 16.97618293762207, -16.9427490234375, 11.538740158081055, -9.385433197021484, -6.238033771514893, 13.370416641235352, 7.455844402313232, 22.396791458129883, 0.25107699632644653, -1.295699954032898, -26.02236557006836], [-11.74790096282959, 7.707037925720215, -28.413330078125, 2.3490188121795654, -31.11336898803711, -24.709436416625977, -0.007599701173603535, -0.5876355171203613, -1.8027551174163818, 13.675302505493164, 0.9676218628883362, -3.662436008453369, 3.5133728981018066, 15.671350479125977, -18.990291595458984, 1.4339399337768555, -9.522175788879395, 36.053253173828125, -1.9573553800582886, 4.5508503913879395, -1.3013941049575806, 4.071549892425537, -2.166902542114258, -6.117292404174805, 8.813017845153809, -1.5140492916107178, -11.109606742858887, 24.653217315673828, -17.845691680908203, -8.285017013549805, -3.3504724502563477, -20.167694091796875, 7.3852105140686035, 8.902092933654785, -21.798620223999023, -22.430696487426758, 9.846542358398438, -20.92889976501465, 1.4712952375411987, 14.946830749511719, 2.978178024291992, -10.455870628356934, 0.8019384145736694, 13.823933601379395, 1.1681764125823975, -15.157405853271484, -28.012237548828125, 5.316765308380127, -0.13309168815612793, -2.697321653366089, 4.04865026473999, 11.34448528289795, -13.442078590393066, -1.0820289850234985, 7.798732757568359, -8.449459075927734, 6.280895233154297, -7.836734771728516, -11.350203514099121, 28.129329681396484, 24.102188110351562, 34.2366943359375, 8.89920425415039, 16.148591995239258], [-15.964780807495117, -10.753262519836426, 20.472440719604492, -4.528659343719482, -1.170802116394043, -8.232291221618652, -12.939249992370605, -13.873908042907715, -25.01089096069336, -4.754802703857422, -15.814505577087402, -26.39261245727539, 3.9888412952423096, -11.116442680358887, 36.89019775390625, -11.696974754333496, 9.648582458496094, -5.70759916305542, -18.613698959350586, -13.436712265014648, -9.896634101867676, 10.663071632385254, -11.269630432128906, -0.6589584350585938, 20.892139434814453, -44.912559509277344, 25.307140350341797, -2.690554141998291, -13.359272003173828, -10.573780059814453, 6.091060638427734, -7.606489181518555, 11.730347633361816, -20.017257690429688, 16.321760177612305, 20.293012619018555, 4.643580913543701, 6.344982147216797, -20.85695457458496, -10.576239585876465, 9.3721923828125, -23.942747116088867, 14.779524803161621, 1.9622058868408203, -2.6867706775665283, -5.310863494873047, -35.12217712402344, -10.474966049194336, 21.99064826965332, 16.79323387145996, 14.682250022888184, -8.61113452911377, -32.155662536621094, -17.474456787109375, -3.2157557010650635, 12.3855562210083, -22.561662673950195, 13.841109275817871, 4.471734046936035, 29.89830780029297, 22.052480697631836, -27.502161026000977, -17.720932006835938, 12.31147575378418], [-1.6907669305801392, -11.507791519165039, -15.480501174926758, 2.9949488639831543, -0.638136625289917, -18.526113510131836, 18.181772232055664, 9.137574195861816, -27.55268096923828, -21.427228927612305, -16.51820945739746, -4.163301944732666, 19.870464324951172, 12.88093376159668, -13.087650299072266, 0.6317646503448486, 13.190510749816895, -13.374594688415527, 13.198137283325195, 9.015056610107422, 34.210811614990234, -2.9816627502441406, -4.826194763183594, -5.1147541999816895, 7.621177673339844, -0.8431704044342041, -5.562160491943359, -37.153076171875, -10.251540184020996, 6.602250576019287, 19.634048461914062, 11.887959480285645, -6.880241394042969, 6.761355876922607, -2.3118319511413574, -2.4946162700653076, -1.9934176206588745, -10.79093074798584, -20.465639114379883, -14.30616283416748, 14.459386825561523, 0.643168568611145, 28.347671508789062, 17.41318702697754, -15.706496238708496, -31.822582244873047, 17.196046829223633, -29.520660400390625, -0.10446356981992722, -8.285927772521973, -19.663484573364258, -15.678013801574707, -9.526089668273926, 14.762933731079102, -17.84389877319336, 10.88824462890625, -16.467077255249023, 13.790897369384766, -0.757574200630188, -19.635034561157227, 31.700714111328125, 7.882288932800293, 47.63396072387695, 2.2015185356140137], [-11.134381294250488, -7.057692527770996, 19.759532928466797, -2.639477491378784, 17.92820930480957, 7.835975170135498, 6.323990345001221, -25.948165893554688, -1.81328284740448, -12.834512710571289, -8.594985961914062, 11.548279762268066, -19.386777877807617, 11.63947868347168, -28.929819107055664, -12.01901912689209, -7.93469762802124, -5.854433059692383, 10.286824226379395, 2.502375602722168, -13.389735221862793, -5.836771011352539, -4.3177714347839355, -22.345035552978516, 3.921903610229492, -3.0748250484466553, -3.53705096244812, -17.22690773010254, -4.575303077697754, -7.109889984130859, -30.206073760986328, -10.60107707977295, 1.514089822769165, 20.983070373535156, -10.489980697631836, 0.899992048740387, -7.90680456161499, 12.24975299835205, 31.688188552856445, -11.79992389678955, -13.941158294677734, -7.964949607849121, 18.26401138305664, 8.86904239654541, -9.339077949523926, -9.082183837890625, 1.364289402961731, 9.86229419708252, -9.959771156311035, 9.902782440185547, -3.4859237670898438, -20.473613739013672, 3.4423532485961914, 5.685090065002441, -22.44911766052246, 26.714702606201172, -19.893936157226562, -21.074623107910156, 2.69541597366333, -3.834786891937256, 7.579867362976074, 28.352825164794922, -3.4779067039489746, 20.580455780029297], [26.82827377319336, 0.10013022273778915, -0.7335508465766907, -1.1923997402191162, 37.79352951049805, 11.478354454040527, -20.333473205566406, -18.042448043823242, 11.542985916137695, 19.31459617614746, -9.38709831237793, 0.4572739005088806, 38.7423210144043, 24.719274520874023, 33.2386589050293, -14.01168441772461, -11.157943725585938, -1.6046347618103027, 5.058741569519043, -16.674339294433594, 10.032423973083496, -15.781305313110352, -5.141152858734131, -0.7175787091255188, 1.1065819263458252, 12.408429145812988, 6.362015247344971, -13.760910987854004, -18.10593605041504, 2.949272871017456, -2.7236506938934326, -12.781253814697266, 5.2591328620910645, 6.445871353149414, -15.655521392822266, -11.193342208862305, -10.809005737304688, 12.94611930847168, -19.446861267089844, -15.301009178161621, -23.33315086364746, 10.274036407470703, -10.582624435424805, 13.84798812866211, -1.1456583738327026, 27.961536407470703, -12.938543319702148, -25.231433868408203, 8.289170265197754, -21.831199645996094, 5.664308547973633, 3.249770402908325, 0.17820563912391663, -12.531343460083008, 7.068584442138672, -3.7355852127075195, -13.009899139404297, 16.05242919921875, -19.970584869384766, 20.686538696289062, 7.449464797973633, -7.914675712585449, -0.8676762580871582, 23.00579071044922], [19.56540870666504, 4.3968048095703125, 9.723602294921875, -12.072220802307129, -1.252543568611145, 16.774484634399414, 32.53620529174805, 12.723074913024902, -26.80234146118164, -18.559505462646484, 18.659029006958008, 14.147247314453125, 8.390666007995605, 29.940750122070312, 3.2793569564819336, 3.135895252227783, -14.329520225524902, -13.452065467834473, -3.0753285884857178, -22.882747650146484, 28.565996170043945, -23.885122299194336, -6.414568901062012, 14.914464950561523, 9.375213623046875, -1.300288438796997, -32.872562408447266, -29.958887100219727, -0.7400043606758118, 23.81754493713379, -14.121089935302734, -22.327346801757812, 13.296538352966309, 2.147409677505493, -27.461030960083008, -8.808366775512695, 3.571807384490967, -19.241212844848633, 12.682628631591797, 15.332385063171387, -16.659746170043945, -28.440195083618164, -15.597512245178223, 6.2491631507873535, -2.602769136428833, 2.478760004043579, -3.2646937370300293, 32.97484588623047, 2.5712645053863525, -18.194320678710938, -24.984355926513672, -9.90399169921875, 5.243690490722656, 12.008281707763672, 16.851104736328125, -26.77765464782715, 27.78077507019043, -33.14519500732422, 4.589571952819824, -8.073019027709961, 6.714877605438232, 8.48105525970459, 14.385581016540527, -21.39099884033203], [7.072539806365967, 14.350274085998535, 37.27094650268555, -6.239604473114014, 26.240093231201172, -7.824185371398926, -17.98174285888672, 13.431388854980469, 4.949162006378174, -12.815388679504395, -45.272220611572266, 15.254528045654297, 22.345046997070312, -13.909242630004883, 8.4661283493042, 2.2059435844421387, -2.248751640319824, -4.816459655761719, 12.541080474853516, -20.33955955505371, 17.23180389404297, -6.336733818054199, -9.400211334228516, -12.377994537353516, -3.3943889141082764, 33.146583557128906, -1.6643953323364258, -3.1974761486053467, 72.2666015625, 0.5043503046035767, 1.9217216968536377, 12.156065940856934, 22.639789581298828, -3.3544819355010986, -15.500731468200684, 9.224068641662598, 0.9612417817115784, -12.19129467010498, 6.130295753479004, -6.855618953704834, 24.579334259033203, -15.267019271850586, 16.2022762298584, 6.77907657623291, -22.36022186279297, 31.603609085083008, 22.838476181030273, 12.907957077026367, -20.205524444580078, 11.634869575500488, -1.2346138954162598, -36.982303619384766, 2.2299787998199463, -23.64290428161621, 40.94021224975586, -6.18865966796875, -16.727487564086914, 1.3976281881332397, -8.16125774383545, -62.895301818847656, -11.917454719543457, -14.600244522094727, -4.408151626586914, 26.358320236206055], [-1.5239861011505127, -9.909878730773926, 12.626608848571777, 11.68021297454834, -5.130269527435303, 6.401416301727295, 24.607994079589844, 5.685849666595459, 2.576399087905884, 31.705177307128906, -9.98637580871582, 7.454252243041992, 8.61685562133789, 11.179500579833984, 2.416741132736206, 10.598465919494629, 4.492598533630371, -27.5262393951416, 22.92079734802246, 3.148524045944214, -5.548426151275635, -13.112859725952148, -15.58401107788086, -48.016300201416016, -28.58170509338379, 20.660781860351562, 14.771496772766113, -20.97239875793457, -12.71918773651123, 18.039735794067383, 25.418235778808594, 36.97740936279297, -15.047504425048828, -7.79921293258667, 17.825233459472656, 7.249418258666992, -5.191471099853516, 41.50196075439453, -15.09760856628418, -17.59331703186035, 18.6851806640625, -38.82058334350586, -23.91073226928711, 5.403765678405762, -13.156259536743164, 21.32628631591797, 18.169498443603516, -17.253009796142578, -0.5383198857307434, -14.785110473632812, 12.168732643127441, -15.584961891174316, -32.05078125, 7.195669174194336, -27.10712242126465, 13.221895217895508, -28.61333656311035, -24.359743118286133, 4.9326605796813965, 6.507991790771484, -25.43152618408203, 17.921836853027344, -7.7219319343566895, 11.512511253356934], [-19.8668270111084, 12.519763946533203, -18.860074996948242, -16.054197311401367, -8.154891014099121, 13.860785484313965, 10.186931610107422, 5.586821556091309, -0.6849172115325928, -22.907575607299805, 4.699310302734375, -31.398300170898438, -3.8679308891296387, -21.827880859375, 8.491790771484375, 18.808895111083984, 19.179946899414062, 31.080081939697266, 4.984316349029541, -7.299437046051025, -1.8993003368377686, 5.894800186157227, 18.395355224609375, 14.987767219543457, 12.683568000793457, 19.436302185058594, 2.3135621547698975, 1.5513501167297363, 7.939354419708252, 8.161540985107422, -20.424325942993164, -4.986815929412842, -12.214066505432129, 1.2703882455825806, 17.343908309936523, 26.695478439331055, 3.3027865886688232, 15.617714881896973, 8.782926559448242, -29.260482788085938, -4.5582098960876465, 1.8643524646759033, -15.944708824157715, 1.9524435997009277, -33.12198257446289, -2.174668073654175, 6.024182319641113, -8.317652702331543, 17.165834426879883, -8.288935661315918, -24.951416015625, -27.042070388793945, -4.394265174865723, 6.591971397399902, -13.837841987609863, 4.057274341583252, 8.440702438354492, -2.4400393962860107, 23.391521453857422, 9.878832817077637, -6.303092002868652, -10.319291114807129, -5.206090927124023, 18.410751342773438], [-9.342375755310059, -15.000247955322266, 18.021203994750977, 17.44329071044922, -5.964426517486572, 19.000041961669922, 28.512990951538086, 55.19054412841797, 3.719243288040161, 1.086485505104065, -1.7998268604278564, 25.249486923217773, -2.527301549911499, 30.540386199951172, -17.201004028320312, -2.1796233654022217, 3.3461484909057617, 14.748823165893555, -4.497703552246094, 11.550972938537598, 2.279733419418335, 11.657386779785156, -14.789917945861816, 6.383928298950195, -22.59674072265625, -9.801547050476074, -6.618967056274414, 19.547119140625, -7.664982318878174, -17.110366821289062, 23.96761131286621, -53.115318298339844, 16.93541145324707, 24.291488647460938, -4.465413570404053, 0.10327353328466415, 20.580018997192383, 28.146038055419922, 34.90827560424805, 27.254375457763672, -24.315935134887695, -15.239930152893066, -5.8879899978637695, -23.077238082885742, 12.3215913772583, 12.26256275177002, -4.910630702972412, 10.329156875610352, -9.738310813903809, 10.879676818847656, -0.9641609787940979, 3.3081276416778564, -14.564696311950684, -0.419706791639328, -7.467793941497803, 14.482763290405273, 13.946529388427734, 4.804046630859375, 3.7317609786987305, -10.478534698486328, 0.47836223244667053, 7.949923038482666, -12.03662395477295, 7.846679210662842], [-13.470160484313965, 2.3692758083343506, -0.10530141741037369, -8.490007400512695, 20.683013916015625, -11.91692066192627, 23.11308479309082, 44.85028839111328, -2.182295560836792, 9.092759132385254, -32.98878479003906, 6.589976787567139, -13.338651657104492, -22.863048553466797, -5.348799705505371, -5.647782325744629, -0.8123533725738525, 21.281219482421875, 17.546306610107422, 8.114982604980469, 18.901470184326172, 25.809682846069336, -34.032196044921875, 29.674816131591797, 22.56223487854004, -1.8987265825271606, 1.8809115886688232, 16.671483993530273, 10.770002365112305, 8.854436874389648, 1.7280831336975098, -9.770071029663086, 0.22869546711444855, 3.2929656505584717, -15.913070678710938, -7.921346664428711, 2.7305424213409424, -3.109029769897461, 7.852706432342529, -1.3095215559005737, 21.71815299987793, 15.947572708129883, 10.75134563446045, 6.952907562255859, -5.4171142578125, -10.251117706298828, -0.6296690702438354, -13.493402481079102, -6.314584732055664, -8.219313621520996, -6.84494686126709, -9.512413024902344, 6.402704238891602, -4.9533562660217285, -8.270809173583984, -29.512054443359375, -5.5471720695495605, -5.927674770355225, -18.727710723876953, 6.194351673126221, -8.104852676391602, -19.89468002319336, 0.004938212223351002, -18.228544235229492]], [[10.700065612792969, -10.418789863586426, -9.937789916992188, 0.07957502454519272, -8.868229866027832, 16.844209671020508, -16.905988693237305, -11.573894500732422, -1.3964385986328125, -1.9385100603103638, 7.346808433532715, -5.0886945724487305, 4.799081802368164, -31.534311294555664, 3.620837688446045, 0.3699849843978882, -19.457225799560547, 32.30044937133789, 28.964794158935547, -12.21422290802002, 15.090808868408203, -5.416576385498047, 11.057548522949219, -4.795656681060791, 11.620515823364258, 3.5735440254211426, 4.3424506187438965, 25.389789581298828, 1.7445000410079956, 33.61199951171875, 12.753718376159668, -3.847421884536743, 7.784121990203857, 0.3564331829547882, -5.1275153160095215, 3.027825355529785, 20.74940299987793, -3.1185712814331055, -2.9507646560668945, -14.469173431396484, -18.297073364257812, -2.217226982116699, 10.365574836730957, 9.850479125976562, 31.504396438598633, 1.4762988090515137, 0.009879122488200665, -14.789576530456543, 20.099912643432617, -7.295469760894775, 1.4176547527313232, 14.264351844787598, 5.959796905517578, -10.470495223999023, -4.540046215057373, 19.140928268432617, -15.983861923217773, 7.485067844390869, 2.4835219383239746, 3.66184139251709, -16.564748764038086, -13.157246589660645, -14.596863746643066, 5.786353588104248], [-3.0341713428497314, 9.049846649169922, -4.231210708618164, -17.70034408569336, -17.74176025390625, -18.18346405029297, 0.7065093517303467, 8.9725923538208, -4.977736473083496, -12.998194694519043, -32.27445602416992, -21.107065200805664, -13.895939826965332, -29.317968368530273, 9.964012145996094, -7.313088893890381, 20.229158401489258, -14.57150936126709, -25.558319091796875, 31.402137756347656, 21.830810546875, -6.542633056640625, -26.56422233581543, 1.2700575590133667, -13.608785629272461, -7.514261722564697, -4.010689735412598, -6.312169551849365, -22.120763778686523, 9.34156608581543, 1.1299859285354614, -6.479724407196045, -4.263624668121338, 15.360013008117676, -19.216691970825195, 16.04067039489746, 9.467638969421387, -10.653986930847168, -8.805482864379883, -21.3717098236084, 2.9021029472351074, -15.837799072265625, -26.811368942260742, 21.322927474975586, 35.053890228271484, 4.550997734069824, 15.610967636108398, 4.194611072540283, 12.27773666381836, -6.3282623291015625, 6.9738311767578125, -27.769643783569336, -17.958086013793945, -22.213172912597656, 17.038402557373047, 14.183280944824219, -12.341241836547852, 4.463432312011719, 2.4769339561462402, 5.6610107421875, -3.8119237422943115, -14.308602333068848, 18.73813247680664, -22.142202377319336], [-11.360860824584961, 11.105278968811035, 11.245478630065918, 2.3419792652130127, -42.209999084472656, 27.263715744018555, 1.8408148288726807, 16.76994514465332, -12.677708625793457, 1.2822290658950806, 15.801033973693848, 15.262999534606934, 8.024178504943848, -18.43662452697754, 18.603206634521484, -0.21025297045707703, -15.320048332214355, 14.185546875, 8.105122566223145, 21.89031219482422, 25.985727310180664, 10.485579490661621, -3.034996747970581, 7.525678634643555, -3.542433500289917, 1.0102992057800293, 13.238140106201172, 23.744417190551758, -7.843419551849365, 10.906116485595703, 19.253582000732422, 5.5486063957214355, -8.72346019744873, 7.723195552825928, -22.231063842773438, -11.959819793701172, 6.941543102264404, 14.044644355773926, 10.807430267333984, 21.291872024536133, 3.7731244564056396, 41.59456253051758, 3.1628358364105225, 35.54514694213867, -8.27795124053955, -0.03952576965093613, 7.635561943054199, -4.278696060180664, 28.417665481567383, 12.12757682800293, -7.198140621185303, -8.5189208984375, -11.447257041931152, 42.20404052734375, 11.547163963317871, -30.142377853393555, -3.973867416381836, 4.283356189727783, -15.641584396362305, 4.872721195220947, 3.5723226070404053, 2.092179775238037, 5.202437877655029, 9.927935600280762], [21.969823837280273, 9.03376293182373, -7.208501815795898, 9.071404457092285, 3.3426575660705566, -6.5398478507995605, -18.561622619628906, 0.3426114618778229, -17.759531021118164, -8.862756729125977, -13.860311508178711, 5.864940166473389, 5.855723857879639, -24.16811752319336, 20.76978302001953, 7.362775802612305, -8.079269409179688, 6.38540506362915, -6.344737529754639, -30.38393783569336, -11.686240196228027, 7.318480014801025, 10.494023323059082, -30.33205795288086, -4.8487396240234375, -10.74833869934082, 15.62833023071289, -29.905261993408203, 3.8420770168304443, -8.970582008361816, -8.610335350036621, -11.561835289001465, -22.583040237426758, -0.5165951251983643, 14.219406127929688, -1.48397696018219, 9.853086471557617, -9.57120132446289, 8.626004219055176, -1.826423168182373, -9.401193618774414, -9.358635902404785, -24.604331970214844, -12.94924259185791, 26.402652740478516, 9.625323295593262, 2.1948118209838867, 0.2702878415584564, -11.532824516296387, -11.968710899353027, 13.572800636291504, -29.03322982788086, 16.539859771728516, 7.531406879425049, 2.944692850112915, -13.920273780822754, 5.894776344299316, 27.01573371887207, 28.513975143432617, -7.111664295196533, 0.7962184548377991, -10.01809310913086, -7.124996662139893, -7.161561965942383], [-2.308797836303711, 12.075277328491211, -18.426116943359375, -30.264732360839844, 1.5726711750030518, 24.373151779174805, 17.7983341217041, -12.356249809265137, 12.560193061828613, -6.825764179229736, -2.8937861919403076, -22.073062896728516, -18.573665618896484, 4.756523132324219, -21.96186065673828, 21.525936126708984, 4.442689895629883, 17.66200065612793, 9.173155784606934, 21.538476943969727, 13.482243537902832, -4.931156635284424, 35.97571563720703, -19.715198516845703, -16.29265022277832, 4.03898286819458, -4.323171615600586, -6.410849571228027, 11.903141021728516, -8.6091947555542, -15.114333152770996, 13.046927452087402, 21.92621421813965, 24.358407974243164, -4.820911407470703, -14.807249069213867, 13.028679847717285, -15.524336814880371, 4.267120838165283, 22.24416732788086, -11.872660636901855, 27.984363555908203, -17.062469482421875, -5.887318134307861, 1.642668604850769, 14.42382526397705, 6.224363803863525, 1.7274744510650635, -3.132079601287842, 3.7785046100616455, 23.833690643310547, -19.175838470458984, -6.829479694366455, 33.06925964355469, -24.235118865966797, -28.479738235473633, 1.499185562133789, 3.9902875423431396, 2.0484399795532227, -22.475027084350586, 5.666735649108887, -19.93539047241211, 10.494403839111328, -0.22233490645885468], [-16.703174591064453, -8.463733673095703, -4.230324745178223, 13.97146987915039, 1.8898340463638306, -16.301116943359375, -12.806266784667969, 7.895593166351318, 33.806217193603516, 21.858680725097656, 17.95023536682129, -1.9069417715072632, 15.135814666748047, -19.164281845092773, -22.316707611083984, -13.404995918273926, -30.97315788269043, 10.611393928527832, 13.451828956604004, -24.03817367553711, 28.528825759887695, 27.141895294189453, 8.069780349731445, -2.003499984741211, 19.241172790527344, -20.7303466796875, -18.911151885986328, 7.164584159851074, 22.274961471557617, 28.38813591003418, -7.153135776519775, -17.932035446166992, 11.748434066772461, -1.9236637353897095, -1.899262547492981, 7.296020030975342, 31.378503799438477, 8.990294456481934, 4.235106468200684, -8.529927253723145, 33.3895149230957, 2.6450064182281494, 21.704824447631836, -5.3809404373168945, -19.141801834106445, -10.454513549804688, 7.388919830322266, 0.4771549105644226, 1.32524573802948, 4.833057880401611, -22.5013427734375, -15.21091079711914, 12.458468437194824, 27.988515853881836, 4.576096057891846, 28.59198570251465, -24.90396499633789, -9.333268165588379, -24.287302017211914, -22.138317108154297, 22.08256721496582, 7.636043071746826, -2.387495517730713, -14.089427947998047], [-1.3563690185546875, -8.770659446716309, -5.686399459838867, -17.306713104248047, 7.416074752807617, -16.498903274536133, 11.134977340698242, -4.379104137420654, -5.279879093170166, 19.759109497070312, -17.117359161376953, 4.97324275970459, -0.4349097013473511, -19.295188903808594, -13.380541801452637, 4.01944637298584, 19.294458389282227, -2.264509439468384, -1.7558664083480835, -12.805334091186523, -1.1822988986968994, -9.487457275390625, -7.967207431793213, -0.9716289043426514, -13.754121780395508, 10.187837600708008, 6.486513614654541, -1.302579402923584, 20.071491241455078, 4.414030075073242, 34.41626739501953, 35.871360778808594, 17.176095962524414, 20.022754669189453, 13.976248741149902, -7.251827716827393, 29.068370819091797, 9.15916633605957, -5.372401237487793, -21.47479820251465, -5.834515571594238, -16.509660720825195, -12.464391708374023, 4.724947452545166, 14.497740745544434, -13.859559059143066, 19.423355102539062, 4.095760822296143, 13.719991683959961, 13.488005638122559, 37.732975006103516, 11.082609176635742, -10.195945739746094, -2.5573372840881348, 16.580028533935547, -7.6444292068481445, -19.852981567382812, 16.212697982788086, -9.137958526611328, -13.706551551818848, 8.373284339904785, 22.577119827270508, -13.391739845275879, -8.419142723083496], [-14.081031799316406, 5.600454330444336, -4.240692615509033, 19.7454891204834, 14.065196990966797, -23.517314910888672, 7.524374008178711, -25.038034439086914, 20.92599105834961, -10.761385917663574, -36.072296142578125, -7.340839862823486, 7.952511787414551, -23.457401275634766, 11.421035766601562, -6.3373003005981445, 13.44393253326416, 0.7496432662010193, -7.734463691711426, -11.553750991821289, 9.01259708404541, -0.5278078317642212, 28.147310256958008, 1.347377061843872, 18.194108963012695, 16.735897064208984, -0.058837197721004486, -7.456990718841553, -21.4305477142334, -0.26497334241867065, -6.5839338302612305, 5.086809158325195, -3.662694215774536, 4.157825469970703, 4.177516460418701, 15.357503890991211, 39.390045166015625, -14.408970832824707, -20.06148338317871, -40.526241302490234, -28.104427337646484, 22.347000122070312, -1.9188493490219116, 1.0323944091796875, 4.931293964385986, -13.538163185119629, 2.3846728801727295, -22.699880599975586, 9.081107139587402, 5.809317111968994, 20.308652877807617, -19.214853286743164, 2.8157691955566406, -19.62008285522461, -24.979379653930664, -6.5249223709106445, -22.381750106811523, 21.452667236328125, 20.725427627563477, -22.925573348999023, -38.42061233520508, -7.507193088531494, 23.19044303894043, 36.4493408203125], [24.087846755981445, -22.926025390625, 7.948824882507324, -9.738582611083984, 21.91761016845703, -6.940261363983154, 13.913263320922852, -4.7647385597229, -3.4658470153808594, 3.287313222885132, -18.02635955810547, 4.155527591705322, -8.239922523498535, 8.139507293701172, -22.123550415039062, 26.622390747070312, -15.104084968566895, 9.71157455444336, 7.118014812469482, 8.030587196350098, -7.986600875854492, 6.33257532119751, -2.9173123836517334, 13.976503372192383, 18.295650482177734, -12.769156455993652, -21.76035499572754, 17.006011962890625, -9.634974479675293, -44.87554168701172, -20.95685577392578, -0.570565402507782, 20.830589294433594, -13.581900596618652, -3.4505679607391357, -6.820578575134277, 17.645402908325195, -2.9538321495056152, 21.962095260620117, -7.597814083099365, 37.57948303222656, -16.22906494140625, -0.48956480622291565, -33.20587158203125, -23.619739532470703, -13.445706367492676, 17.581317901611328, -4.052272319793701, -23.855897903442383, -28.861257553100586, 4.363548755645752, -6.456526279449463, 20.2654972076416, 9.884175300598145, -36.85041809082031, 9.813421249389648, -14.070056915283203, 14.93173599243164, 5.799736976623535, 24.015438079833984, -16.786725997924805, 10.845571517944336, -10.006630897521973, 9.0308198928833], [-3.684115171432495, 24.1070613861084, 19.5985107421875, -3.9586269855499268, -16.869539260864258, -14.802342414855957, -1.3019613027572632, -3.803041934967041, -1.1575965881347656, 16.310941696166992, -9.093398094177246, 15.766704559326172, -8.298112869262695, -29.630413055419922, -15.86241626739502, 1.492672085762024, -7.861547946929932, -37.93819046020508, -12.94168472290039, 8.342411041259766, 4.775280952453613, -9.729328155517578, -1.3113529682159424, -6.129091262817383, -5.873798847198486, -31.62538719177246, -4.907666206359863, -22.437774658203125, 6.609377384185791, 24.83043670654297, 21.53793716430664, -18.76894187927246, 6.025920391082764, 19.011350631713867, -0.20951759815216064, -0.38157179951667786, 26.44574737548828, 1.73210871219635, 26.346952438354492, -10.815114974975586, 6.595741271972656, -2.2276999950408936, -2.3975391387939453, 29.89771842956543, 21.434619903564453, -10.917494773864746, 1.469536542892456, 7.264376640319824, 13.654213905334473, 4.849850654602051, -8.124460220336914, 2.826476573944092, -1.7266350984573364, -6.422731399536133, 0.5713818073272705, -13.780074119567871, -29.94248390197754, -1.0240055322647095, -11.907829284667969, -10.495960235595703, -8.492467880249023, 17.587228775024414, -0.2813519239425659, -0.9496201872825623], [16.313936233520508, 8.023100852966309, 18.06462287902832, 1.217397689819336, -6.515080451965332, 47.2340202331543, -1.8576411008834839, 8.929160118103027, 13.461580276489258, -5.6221394538879395, -41.98588943481445, 14.994114875793457, -4.11129903793335, 37.579593658447266, 28.850547790527344, 6.562712669372559, 6.411523818969727, 22.890403747558594, 22.171056747436523, -7.831517219543457, 61.57989501953125, -2.7806460857391357, -13.765396118164062, -10.121685981750488, 15.336915016174316, 5.048776626586914, 6.430500030517578, 10.482650756835938, -5.443329811096191, -6.580148220062256, -19.641443252563477, -1.4554407596588135, -23.604516983032227, 25.553028106689453, -0.7541027665138245, -1.5192856788635254, 8.831008911132812, -16.60818862915039, -4.037881851196289, 14.585809707641602, 1.8552875518798828, 7.981557846069336, -16.678865432739258, -7.249587059020996, 12.862418174743652, 10.067484855651855, 16.682580947875977, -27.00090980529785, 15.076554298400879, -10.613005638122559, 1.7233704328536987, -4.2746148109436035, 2.6768686771392822, 3.6697211265563965, -6.574565410614014, -5.186481952667236, -8.154038429260254, 0.6887964606285095, 17.586984634399414, 27.269912719726562, -23.100561141967773, -22.508365631103516, 7.264413356781006, -5.652638912200928], [2.3904025554656982, 11.179849624633789, -12.60242748260498, -4.266668796539307, 19.890331268310547, 5.739275932312012, -9.684876441955566, -9.203346252441406, 12.754592895507812, -2.27961802482605, 39.88726806640625, -10.1611328125, -0.4113885164260864, -4.7137227058410645, 23.809062957763672, -12.335297584533691, 9.728602409362793, 14.071648597717285, 21.239425659179688, 27.168701171875, -3.6346595287323, -11.349377632141113, 8.736540794372559, 8.05095100402832, 21.232603073120117, -11.886272430419922, 5.396346569061279, -1.507738471031189, -12.84782886505127, -12.27791690826416, -8.420923233032227, -2.4871058464050293, 0.842088520526886, -17.889907836914062, 26.56212615966797, 10.614387512207031, -10.057138442993164, 17.615846633911133, -10.558974266052246, -0.9982196688652039, 11.223553657531738, 14.37486457824707, -28.159706115722656, 6.399913787841797, 1.3687883615493774, 11.938478469848633, -5.306263446807861, -8.836774826049805, -3.1367950439453125, -10.774435043334961, -18.340593338012695, -12.543839454650879, -21.439266204833984, -7.732320308685303, -11.991850852966309, -3.0766780376434326, 32.0018424987793, -1.4947515726089478, 10.308652877807617, 3.6629812717437744, 6.330073833465576, -12.79604434967041, 14.223098754882812, 15.581713676452637], [-9.895386695861816, -26.5874080657959, -13.507791519165039, 16.40236473083496, 6.494118690490723, 4.523629188537598, -3.6606175899505615, 16.710498809814453, 10.915945053100586, 8.514571189880371, -4.78278112411499, -35.45134353637695, -5.147340774536133, -3.5097029209136963, -23.044347763061523, 12.12602424621582, -24.9976749420166, 12.558487892150879, -7.723110675811768, 12.891894340515137, -4.408236980438232, 15.253291130065918, -4.403209686279297, 3.167058229446411, 8.672867774963379, -8.772400856018066, -5.609678268432617, 20.474533081054688, 3.0555315017700195, 16.983720779418945, 9.987215995788574, 38.06266784667969, 18.030071258544922, -41.733924865722656, 12.160676002502441, 7.669055461883545, -2.8547732830047607, 7.478572368621826, -3.8753902912139893, 5.2469258308410645, 13.436251640319824, 10.640213966369629, 15.815170288085938, 22.03291893005371, -3.0540611743927, -2.470374345779419, -23.138029098510742, 5.576122283935547, 6.661643028259277, 12.316449165344238, 24.6411190032959, -15.562037467956543, -6.036591529846191, -12.344493865966797, -5.475854396820068, 13.621613502502441, 3.208930015563965, -21.30038070678711, -23.277551651000977, -10.614411354064941, -12.301950454711914, 6.200313091278076, -0.6202789545059204, -33.17677307128906], [1.5133652687072754, -8.291001319885254, 5.627100944519043, 24.569255828857422, 32.0742073059082, -45.23752212524414, 1.5494102239608765, 15.02737045288086, -13.310097694396973, -27.18820571899414, 8.090389251708984, 13.8816556930542, -8.893701553344727, -5.020665645599365, 1.161732792854309, 12.658452033996582, 30.94216537475586, 14.337474822998047, -23.820249557495117, -21.841577529907227, -15.742774963378906, -8.248970985412598, 6.616257667541504, -4.822979927062988, -4.866469860076904, 3.575803279876709, 13.061796188354492, 3.525658369064331, -7.80615758895874, 16.681379318237305, -15.889447212219238, 7.158236026763916, 11.398649215698242, -27.998737335205078, 1.2830857038497925, -8.93055534362793, -27.821792602539062, 18.932809829711914, 37.84297561645508, -17.153148651123047, -18.77131462097168, -28.339279174804688, -8.215485572814941, 20.335708618164062, 5.175617694854736, -3.592663049697876, 7.079969882965088, -42.972564697265625, -12.501265525817871, 27.522735595703125, 10.459230422973633, 27.099262237548828, -14.43903923034668, -18.36381721496582, 3.4851322174072266, 9.271021842956543, -4.257389545440674, 23.5584716796875, 16.047651290893555, 25.16279411315918, 9.2156982421875, 15.066238403320312, -40.61122131347656, 12.127045631408691], [17.196643829345703, -22.29654884338379, -25.820049285888672, -21.757505416870117, -13.744452476501465, -3.7806408405303955, 1.2144452333450317, 16.028343200683594, -10.325514793395996, 16.453330993652344, 24.435474395751953, 24.5648193359375, 6.621143341064453, -22.496349334716797, 2.4494779109954834, 11.876741409301758, 1.8963342905044556, -16.126317977905273, -3.1524152755737305, 8.801461219787598, -9.678744316101074, 0.0059642670676112175, 14.518572807312012, -8.048691749572754, -4.958411693572998, 22.160388946533203, -24.41674041748047, 1.6742340326309204, 8.16948413848877, -9.056832313537598, -3.381901741027832, 10.46292781829834, 10.113835334777832, -3.1528279781341553, -10.144960403442383, 0.36299973726272583, 4.980310440063477, 19.123340606689453, 4.997420310974121, 4.546047210693359, 11.54546070098877, -2.364800214767456, -3.338395833969116, -2.2241017818450928, 3.9899117946624756, 1.79079270362854, -5.805239677429199, -10.755110740661621, -0.3463456630706787, -17.445161819458008, -19.870418548583984, 6.10161018371582, -0.368174284696579, 8.270573616027832, 3.5810415744781494, 32.125709533691406, -11.343868255615234, -14.74439811706543, -17.6464900970459, 22.29645538330078, 9.852046966552734, 4.397950172424316, -3.1179351806640625, 24.659278869628906], [11.75959300994873, -9.136794090270996, 3.4188215732574463, 8.720200538635254, -15.060785293579102, -15.30300235748291, 15.064555168151855, -9.287620544433594, -16.90395736694336, -2.4827325344085693, -2.2097997665405273, -6.785464763641357, 19.161611557006836, -14.192187309265137, -21.107412338256836, 2.5787277221679688, 26.695323944091797, -19.27000617980957, -0.9290608167648315, -14.79652214050293, 4.510014057159424, 0.3367198407649994, 10.312996864318848, -8.172751426696777, -10.254084587097168, -1.941720962524414, 7.3227219581604, 14.25674057006836, 34.42249298095703, 14.487481117248535, 21.276758193969727, 19.513036727905273, -11.492203712463379, -3.081049680709839, -11.801410675048828, -28.971391677856445, -5.302163124084473, 11.589323043823242, 6.620606422424316, 14.868528366088867, 32.902992248535156, 12.41123104095459, -18.095516204833984, -0.11808598786592484, -27.434429168701172, 4.782499313354492, 5.657534599304199, -33.070274353027344, -3.217738389968872, 0.7419376373291016, 6.617201328277588, 18.238903045654297, 4.436529159545898, 4.375909328460693, -1.8583810329437256, 23.032772064208984, -1.2371289730072021, -10.486912727355957, 12.956267356872559, 8.419511795043945, 21.249448776245117, 35.61800765991211, 5.913571357727051, 29.24445915222168]], [[17.432167053222656, -19.37943458557129, 9.957005500793457, -13.194989204406738, 0.9601014852523804, 17.32999038696289, 2.631714105606079, -21.944015502929688, 7.859842777252197, -18.218734741210938, 11.554944038391113, -5.071658611297607, -21.21514320373535, -11.980443000793457, -28.132768630981445, 12.745612144470215, -0.8831470608711243, -2.8298516273498535, 4.1301374435424805, -2.662703275680542, -31.437557220458984, 3.9679393768310547, 14.02461051940918, 5.010342597961426, 4.24974250793457, 17.40065574645996, -26.61104393005371, -19.33552360534668, 2.058920383453369, -5.574172496795654, -10.158489227294922, -11.683796882629395, -21.224807739257812, 7.975151062011719, -4.20047664642334, 17.873332977294922, 20.084897994995117, -36.6658935546875, 20.494020462036133, 11.78130054473877, 4.777111530303955, -9.685328483581543, -0.33239996433258057, -12.13579273223877, 10.275185585021973, -0.18280479311943054, -2.5319457054138184, 5.7162699699401855, -6.075384616851807, -13.579432487487793, 7.410661220550537, -31.064626693725586, -26.39630889892578, 25.84488868713379, -9.435454368591309, -6.1799139976501465, -27.01797866821289, 12.742286682128906, 21.07724380493164, 20.827714920043945, -13.793466567993164, 11.832812309265137, -5.007137775421143, 2.098954916000366], [-19.91309356689453, -18.990528106689453, 8.605888366699219, -28.60965919494629, 0.4658741056919098, -7.477376937866211, 7.098955154418945, 2.2983570098876953, 9.02696704864502, 24.991392135620117, -12.771394729614258, 0.7001911997795105, -8.158065795898438, -5.740381717681885, -12.981302261352539, 29.830307006835938, -22.147119522094727, -13.543206214904785, 9.565986633300781, -15.242146492004395, -31.587116241455078, -18.88627815246582, 4.560330867767334, 2.0940942764282227, -4.746777534484863, 5.946534156799316, -2.4785454273223877, -5.302881717681885, 1.1156526803970337, 12.971078872680664, -11.49406623840332, -14.74036979675293, -16.646759033203125, -20.92485809326172, -8.652212142944336, -4.4172587394714355, -0.8925357460975647, 14.522383689880371, 11.525877952575684, -0.07739690691232681, 3.6686058044433594, 16.2127628326416, 25.837121963500977, 0.43807339668273926, 5.443445682525635, -9.447796821594238, -24.29195213317871, -16.520544052124023, 6.700320243835449, -21.91270637512207, -2.1283838748931885, -12.997482299804688, 14.686421394348145, -3.0611157417297363, 23.2669734954834, -15.84172534942627, 17.525392532348633, -17.608354568481445, -18.492448806762695, 6.09922981262207, 1.1603517532348633, -24.225122451782227, -1.0621602535247803, -23.081161499023438], [-31.882598876953125, 1.081451177597046, 7.5370001792907715, -25.553768157958984, 21.825979232788086, 1.8215245008468628, -9.406083106994629, 35.568077087402344, -7.765219688415527, -4.51693868637085, 0.5249137878417969, -1.4670828580856323, -14.075620651245117, -16.471363067626953, 6.384746074676514, 15.688756942749023, -51.40376663208008, 1.014782428741455, -2.2301182746887207, 17.41715431213379, -2.493004083633423, 14.96181869506836, -16.50244140625, -2.387784719467163, 27.565998077392578, -17.7098331451416, -9.804832458496094, 29.09232521057129, -6.623866081237793, -13.108139991760254, -0.2504623234272003, 20.215967178344727, 17.517457962036133, -10.469143867492676, 5.322689056396484, 15.857614517211914, 4.712096214294434, -7.5256452560424805, -2.418877363204956, -10.285770416259766, 26.169065475463867, -8.087077140808105, 9.69906234741211, 5.642855644226074, -14.73501205444336, 0.5292261242866516, -3.4110937118530273, 21.663543701171875, 15.523465156555176, -14.05169677734375, 18.459001541137695, 25.91830825805664, 3.3414981365203857, 6.0309834480285645, -11.369338035583496, -0.31423208117485046, 23.875816345214844, 23.34712791442871, 10.15938949584961, 8.184380531311035, -4.456568717956543, -12.832448959350586, 22.248626708984375, -18.614595413208008], [3.2709922790527344, -13.58444595336914, -7.690499305725098, 34.99393844604492, 17.84052848815918, -13.73256778717041, 30.331130981445312, -20.604639053344727, -17.691429138183594, 14.768668174743652, -8.623746871948242, 12.918312072753906, 6.502533435821533, -7.817703723907471, -3.0078494548797607, 7.505389213562012, 2.736119031906128, 2.0580430030822754, 5.722589492797852, 31.448856353759766, -16.377723693847656, -40.43596267700195, -13.292409896850586, -5.893848896026611, -27.074451446533203, 5.120442867279053, -11.351858139038086, 13.253744125366211, -7.040350914001465, 35.12632369995117, -7.904633522033691, 7.725101947784424, 9.476823806762695, 9.820897102355957, -16.156911849975586, 29.410558700561523, -2.980010986328125, 17.75641441345215, -2.9337656497955322, 19.924524307250977, -7.356524467468262, 11.832136154174805, -11.086127281188965, 19.994293212890625, 3.2239131927490234, -24.95680809020996, 15.840571403503418, 14.586274147033691, -13.06273365020752, -29.30054473876953, 7.773280620574951, 8.087045669555664, -6.757440090179443, 2.416240930557251, 3.515285015106201, 11.192387580871582, 14.859479904174805, -19.351293563842773, 4.3327226638793945, 15.10489273071289, 14.006115913391113, -9.61011791229248, -7.687280178070068, -3.2587625980377197], [-14.358894348144531, 19.45771026611328, 13.464609146118164, -19.978740692138672, 11.580092430114746, -6.312291622161865, 4.924375057220459, -22.5364990234375, -33.80539321899414, 1.1964284181594849, -4.252584457397461, 8.862979888916016, -4.174709796905518, 5.06878137588501, 1.7250062227249146, -3.130131244659424, -1.113696813583374, -2.317840814590454, -9.047846794128418, 2.1927592754364014, 50.83830261230469, -20.072458267211914, 21.985374450683594, -1.8325403928756714, -2.500375986099243, 19.55975341796875, 10.955055236816406, 14.908682823181152, 3.3193700313568115, -19.996164321899414, -1.892876148223877, 12.045591354370117, -7.555297374725342, 1.8149276971817017, -11.056302070617676, 18.814329147338867, -17.52135467529297, 20.482295989990234, 14.660233497619629, 4.732946395874023, -17.21503257751465, 14.217547416687012, 20.912757873535156, -4.634706020355225, 9.946846961975098, -8.762410163879395, 30.909894943237305, 5.195131301879883, 16.63726806640625, 4.181863784790039, 31.268842697143555, 28.93240737915039, -11.447714805603027, 4.426821231842041, -8.977173805236816, 14.139510154724121, 9.317455291748047, 12.263959884643555, 14.161795616149902, -3.403883695602417, -3.425710678100586, -11.26013469696045, 12.68569564819336, 6.525443077087402], [-23.141115188598633, 14.536937713623047, -14.067220687866211, 1.2150704860687256, 0.020628899335861206, -6.751161575317383, -7.609071254730225, -22.66004180908203, 10.227447509765625, -0.23063045740127563, 18.3045597076416, 19.705228805541992, -13.356155395507812, 14.089519500732422, -7.400140762329102, -35.36244583129883, 0.9977154731750488, -1.9958093166351318, -4.291049480438232, 3.4424333572387695, -21.209577560424805, 21.987924575805664, -11.44008731842041, 5.72098970413208, -5.989436626434326, -29.84259605407715, 26.070554733276367, -9.083401679992676, 3.441037178039551, -13.715080261230469, 7.883021831512451, 0.5337765216827393, 1.9743024110794067, -29.95362663269043, -3.0140538215637207, 3.8153817653656006, 6.852787017822266, -8.9100341796875, -12.057754516601562, 14.86240291595459, 1.9190059900283813, 21.967510223388672, -14.153636932373047, 2.478757381439209, 11.560277938842773, 4.27858304977417, -1.6574596166610718, -11.774944305419922, -16.70798683166504, -14.04694938659668, -4.036996364593506, -4.103106498718262, 3.9144558906555176, -1.6439216136932373, -0.9424887299537659, -24.033550262451172, 18.561946868896484, 2.549821615219116, 15.398547172546387, -3.6332759857177734, -15.806392669677734, 13.005345344543457, -3.740476369857788, 10.471370697021484], [3.7398929595947266, 3.0677199363708496, -12.461316108703613, 9.193392753601074, 19.988325119018555, -5.424190998077393, -6.519942283630371, -3.591144323348999, -6.239758491516113, 13.042033195495605, -11.622976303100586, 20.089033126831055, -10.312187194824219, 10.735502243041992, -10.389786720275879, -40.43398666381836, -2.710718870162964, 24.866193771362305, 20.88600730895996, 9.017183303833008, 23.941099166870117, 4.814701557159424, -2.5215742588043213, 5.849758625030518, 2.372434139251709, -2.9170329570770264, 36.82615280151367, 2.468501091003418, -2.0763742923736572, 3.3928377628326416, -3.2270326614379883, 15.917130470275879, 17.606719970703125, 5.967988967895508, -10.86498737335205, 23.365154266357422, 11.967945098876953, -10.78423023223877, 15.51834774017334, -23.20258140563965, -14.74092960357666, 12.814078330993652, -7.916894435882568, -15.21245002746582, 3.0219318866729736, -28.622358322143555, 2.145556688308716, -16.734403610229492, 30.47217559814453, -11.076539039611816, 22.91745948791504, 34.05569839477539, 1.0470319986343384, -11.155317306518555, -2.7638821601867676, -0.6346752047538757, 3.173096179962158, -4.331905364990234, 10.985983848571777, -21.905370712280273, -23.07645034790039, -9.561217308044434, 19.717525482177734, 10.483291625976562], [1.9315913915634155, -21.901952743530273, 5.012688159942627, 4.294397830963135, -1.2845933437347412, -9.064902305603027, -13.180023193359375, 17.16728401184082, 25.147762298583984, 6.486026763916016, -22.892608642578125, -2.4926650524139404, 9.898555755615234, -22.15804100036621, -44.08536148071289, 26.029747009277344, -36.38737106323242, -7.547210693359375, 9.8870849609375, 1.4835761785507202, -15.402156829833984, 6.921401023864746, -14.363889694213867, -6.90939474105835, 25.01729965209961, -13.749076843261719, 16.70932388305664, -14.62588119506836, -7.625418186187744, 23.781747817993164, 25.320117950439453, 13.067161560058594, -12.709697723388672, -5.947461128234863, -2.5762548446655273, 4.587337970733643, 1.0283024311065674, 1.8590290546417236, -9.867620468139648, 18.89910125732422, -5.144274711608887, 39.512939453125, 7.354339122772217, -14.253151893615723, 2.9981331825256348, -16.78063201904297, 35.73185348510742, -4.990746021270752, 1.7200185060501099, -2.8622119426727295, -41.43642807006836, 20.055587768554688, 2.304318428039551, 3.828129768371582, -8.740376472473145, -0.6518924236297607, 36.095863342285156, 10.020512580871582, -17.589067459106445, 11.330760955810547, 2.61238956451416, -27.996076583862305, 8.706348419189453, 3.5322067737579346], [4.759387969970703, -10.473551750183105, 1.2316720485687256, -4.836365699768066, -30.285470962524414, 15.545948028564453, -3.9504878520965576, -22.110889434814453, 23.502710342407227, -21.192567825317383, 4.305124759674072, 3.8735132217407227, 6.039919376373291, -7.617382049560547, 25.88843536376953, -13.220412254333496, -15.637916564941406, 8.467981338500977, 7.346384048461914, 10.229872703552246, 10.161843299865723, 0.48566582798957825, 13.218954086303711, -3.382488250732422, 23.074298858642578, -8.143898010253906, 2.731182336807251, 6.628749370574951, -4.8527140617370605, -34.71936798095703, 13.037154197692871, 1.8807052373886108, 10.4058198928833, -22.67026138305664, -17.201393127441406, -15.14731502532959, 2.7914650440216064, 8.581936836242676, -2.1089227199554443, -15.304732322692871, -1.1691495180130005, 11.89097785949707, 6.351906776428223, -4.9778852462768555, -7.2253546714782715, 27.832494735717773, 12.050734519958496, 6.009969234466553, -9.03372573852539, -0.29931703209877014, -9.119678497314453, -7.530534744262695, -23.61142921447754, -28.149486541748047, -0.9970341324806213, -9.763750076293945, -15.425908088684082, 15.147729873657227, -16.192407608032227, -13.183308601379395, -2.6450769901275635, -10.904624938964844, -3.113818407058716, 9.010900497436523], [39.313961029052734, 10.337166786193848, -25.354047775268555, 5.06154727935791, 1.4590873718261719, -21.594314575195312, 28.269607543945312, 26.373638153076172, -15.972761154174805, -4.581002712249756, -13.01118278503418, 4.29376220703125, 20.831594467163086, 11.69543743133545, -4.652525901794434, 16.655824661254883, 26.941743850708008, 6.625253677368164, 14.06461238861084, -0.028995294123888016, 29.702423095703125, -6.5165252685546875, 12.191843032836914, 11.14210033416748, -1.5757805109024048, 7.3967604637146, 9.503786087036133, -33.475032806396484, -10.019584655761719, 19.274761199951172, 6.163543224334717, -3.198451280593872, -36.3578987121582, 15.608007431030273, -4.08355712890625, -17.292522430419922, 7.514398574829102, -8.651121139526367, -36.081382751464844, 30.078453063964844, 3.86357045173645, 22.13191032409668, 2.756765127182007, 4.198124885559082, 2.27546763420105, 5.150845527648926, 19.764812469482422, 2.8934237957000732, 32.68887710571289, -0.37020277976989746, -6.379901885986328, -29.10017204284668, -8.900079727172852, 17.342716217041016, 13.515347480773926, 27.303565979003906, 35.6763801574707, 20.75957679748535, 4.506308078765869, 12.821279525756836, 4.694853782653809, -14.81580638885498, 27.966888427734375, -26.436809539794922], [2.1526846885681152, -23.581769943237305, 0.715190589427948, -0.8063547015190125, 31.41739845275879, -32.34778594970703, 10.367792129516602, 26.71222686767578, -4.9050750732421875, 26.299911499023438, -1.9555472135543823, -8.318427085876465, 6.307448863983154, 9.437995910644531, 6.15255069732666, 11.99718189239502, -5.923212051391602, 15.970527648925781, 7.504851818084717, 7.805546760559082, 22.36229705810547, -18.686288833618164, 6.879323959350586, -17.239988327026367, 12.694478034973145, 8.72492790222168, 3.5356602668762207, -14.339197158813477, 24.1716251373291, -5.56450891494751, 11.632479667663574, -10.700549125671387, -6.277011871337891, 5.742396831512451, 7.664161205291748, 7.657076835632324, -11.61128044128418, -5.142502784729004, 6.617571830749512, -24.846799850463867, -13.241713523864746, 16.28537940979004, 16.963289260864258, 15.629947662353516, 9.311723709106445, 3.920294761657715, -24.562646865844727, -10.153775215148926, 19.10992431640625, -32.953819274902344, 2.6435747146606445, 9.598908424377441, -13.431131362915039, -0.8508466482162476, 15.448164939880371, 8.09957504272461, -5.222270488739014, -11.82595157623291, 0.31819668412208557, -6.208641052246094, 0.8679458498954773, -3.5647976398468018, 15.97488784790039, 16.09373664855957], [13.487882614135742, 0.577120304107666, -20.66119384765625, -3.572007894515991, 6.154174327850342, 1.4194294214248657, 4.195079326629639, -0.21133564412593842, 7.696071147918701, 5.052827835083008, -24.610443115234375, 22.054723739624023, -37.43852233886719, -11.091422080993652, 8.268625259399414, 6.025824069976807, 15.176006317138672, 21.194856643676758, 12.76303482055664, -4.028594017028809, -11.270383834838867, -16.23731231689453, 13.385504722595215, -18.125967025756836, -15.603403091430664, -13.769874572753906, 8.24319076538086, 16.007774353027344, 13.244733810424805, 7.9498138427734375, 13.163765907287598, -33.36890411376953, 9.379533767700195, 5.020509719848633, 3.74735689163208, -6.448873996734619, 13.811995506286621, 7.865719795227051, -6.778635025024414, 0.6938319802284241, 1.2748253345489502, 4.406641960144043, 2.407832145690918, 21.553936004638672, -21.210792541503906, 22.927061080932617, 8.167895317077637, -14.668437004089355, 2.3241937160491943, -13.14665412902832, -24.660320281982422, -23.496856689453125, -21.827373504638672, -23.69156265258789, -2.8061249256134033, -16.982784271240234, 14.905945777893066, -12.390493392944336, -31.191993713378906, -21.108816146850586, -0.9633380174636841, -11.675495147705078, 13.178616523742676, 18.554445266723633], [-8.821828842163086, -1.0877482891082764, -18.10460090637207, -15.518989562988281, 3.7880656719207764, 24.545738220214844, 8.524048805236816, 17.512371063232422, -20.182621002197266, 25.38022804260254, 0.006112660747021437, 24.526508331298828, 13.439338684082031, -14.712128639221191, 29.64926528930664, 7.876218318939209, -11.509129524230957, -10.521564483642578, 8.969206809997559, 8.83747386932373, 11.511993408203125, -7.743191242218018, -2.324087142944336, 8.365861892700195, 3.915491819381714, 8.698153495788574, 6.646886825561523, -8.664148330688477, -14.635322570800781, 3.108381509780884, -13.468992233276367, -23.37398338317871, -4.602878570556641, -9.735857963562012, 3.968043327331543, -2.409827709197998, 10.968193054199219, -7.4832563400268555, 17.117597579956055, -28.650949478149414, -8.57918643951416, -3.6810288429260254, -18.564207077026367, 27.82803726196289, 2.5529086589813232, 6.714941501617432, 23.470489501953125, 21.825599670410156, 2.203021764755249, 32.741058349609375, -17.7545108795166, 8.235814094543457, -3.7138071060180664, 27.775943756103516, 2.980591297149658, -4.599603652954102, 26.371644973754883, 41.64455795288086, -7.1626763343811035, -18.544448852539062, -12.810199737548828, 6.705381870269775, 2.136967420578003, 10.442585945129395], [2.8103673458099365, 3.1228063106536865, -11.102777481079102, -10.575495719909668, -1.136303186416626, -8.335855484008789, -15.799665451049805, 20.13475799560547, -7.910676002502441, -10.416650772094727, -9.40512752532959, -11.6151123046875, -8.3904390335083, 24.17612075805664, 12.158966064453125, -31.231555938720703, -20.39067268371582, -35.81986999511719, 1.1715056896209717, 6.068028450012207, 9.485686302185059, 7.050414085388184, -23.678020477294922, -7.095565319061279, 14.651459693908691, 0.47851163148880005, 16.64434814453125, -9.052948951721191, -3.1269829273223877, 4.729607582092285, -1.3720734119415283, -6.630660057067871, 10.502400398254395, 2.2408030033111572, -33.74660110473633, 2.3539249897003174, 8.743370056152344, 4.126266002655029, -27.318893432617188, -10.199548721313477, -5.801405429840088, 18.91150665283203, -14.900519371032715, 0.5404975414276123, 16.60640525817871, -1.038484811782837, 9.710958480834961, -14.462584495544434, -12.007061004638672, 11.793170928955078, -16.0377140045166, 8.701095581054688, 5.07372522354126, 5.887923240661621, 10.106186866760254, 3.434586763381958, 10.962483406066895, -30.3896427154541, 11.964350700378418, -18.620718002319336, -17.033838272094727, 7.490004062652588, 12.601824760437012, 3.8243324756622314], [-8.467132568359375, -5.688320636749268, -15.471614837646484, -19.327877044677734, -18.654756546020508, -6.5243964195251465, -9.650328636169434, 3.887620210647583, -20.867904663085938, -5.0109639167785645, -32.70142364501953, -21.846664428710938, -9.556449890136719, -11.163969993591309, 9.455196380615234, -28.779985427856445, 0.43216636776924133, -20.87854766845703, 29.47907257080078, -8.224374771118164, -6.93766450881958, 19.24401092529297, -8.502111434936523, -10.772689819335938, 8.33647346496582, -10.18191909790039, 10.357842445373535, -15.764595985412598, -10.826532363891602, 4.160866737365723, -14.843585014343262, -16.2600040435791, -49.28607177734375, 24.69473648071289, 21.655431747436523, -5.737964630126953, 10.993802070617676, 24.841203689575195, -4.550080299377441, -25.9574031829834, -10.625250816345215, 11.681564331054688, -9.705821990966797, -13.20700454711914, 2.543750286102295, -6.03886079788208, 17.81436538696289, -4.028980731964111, -14.953551292419434, 11.50759506225586, -5.515142917633057, -12.750578880310059, 1.5841604471206665, 0.7338253855705261, -1.7563402652740479, 31.713327407836914, 5.308579921722412, 19.225072860717773, 6.011908531188965, 20.080820083618164, 8.392160415649414, -16.636568069458008, -0.29437056183815, -6.443718910217285], [-1.0015106201171875, -6.076427936553955, -18.214683532714844, -15.56268310546875, -17.178539276123047, -8.35638427734375, -18.35173225402832, -10.908651351928711, -28.867464065551758, -1.4991756677627563, -20.731048583984375, 26.28396987915039, -0.08612906187772751, 33.483551025390625, 8.180144309997559, -1.7772772312164307, -16.568153381347656, 8.470010757446289, -8.04915714263916, -27.324007034301758, 10.41670036315918, 4.7354888916015625, 26.54277229309082, 3.1600444316864014, 24.522838592529297, 5.42509126663208, -4.527586460113525, 15.629413604736328, -4.15495491027832, -2.499826669692993, -26.859004974365234, 17.62383460998535, 14.514328002929688, -19.64334487915039, -0.44473081827163696, 1.0465855598449707, -9.43661880493164, -13.093143463134766, -9.236875534057617, -8.190378189086914, -19.92817497253418, 18.00408172607422, -9.117508888244629, 6.943994998931885, 4.1823883056640625, -25.358985900878906, 16.271780014038086, 8.17499828338623, -6.995561122894287, -0.491987019777298, 7.363734722137451, 1.3746709823608398, 9.979962348937988, 0.17081162333488464, 20.415573120117188, -12.547365188598633, 23.279741287231445, 7.391141414642334, 13.650968551635742, -39.71210479736328, -15.000265121459961, 2.7325377464294434, 11.266236305236816, -5.174416542053223]], [[-11.498132705688477, -27.642427444458008, 2.13232159614563, -0.4379726052284241, 8.697530746459961, 15.770834922790527, 7.368139743804932, 12.235384941101074, -17.660484313964844, 18.25323486328125, -11.433303833007812, -2.0831165313720703, -4.379288673400879, -19.138385772705078, -34.353553771972656, 17.01769256591797, -23.841941833496094, 11.515739440917969, 4.089469909667969, 15.447563171386719, 4.9353461265563965, -9.381175994873047, 6.161339282989502, -10.222262382507324, 13.665600776672363, 8.409126281738281, 30.443445205688477, 26.437847137451172, 4.029546737670898, 3.741486072540283, -2.1857433319091797, 8.64942455291748, 16.394725799560547, 17.49343490600586, 10.094961166381836, 28.448139190673828, 4.029777526855469, -10.025811195373535, 5.52518367767334, 9.274138450622559, 1.0485066175460815, 17.737672805786133, -13.003181457519531, -10.166589736938477, -3.9592080116271973, 5.682497501373291, 20.475385665893555, -3.2790591716766357, -16.923620223999023, -9.661508560180664, 3.982557773590088, 7.176212310791016, 46.667701721191406, 10.694786071777344, -21.870777130126953, -7.5330915451049805, -19.6806697845459, 0.4516548812389374, -2.4824767112731934, -1.769826889038086, 9.601370811462402, -32.17948532104492, 4.700068473815918, 0.014872937463223934], [15.428645133972168, 17.12127113342285, -1.6970065832138062, -22.940040588378906, 16.848751068115234, 4.664091110229492, -5.041741847991943, 36.39786911010742, -29.393938064575195, 24.15433120727539, -20.08586311340332, -14.963655471801758, -8.669368743896484, -9.140616416931152, -0.44165417551994324, -11.065658569335938, -19.843353271484375, 31.397974014282227, -5.114297866821289, -22.413284301757812, -15.979748725891113, -18.19162940979004, -16.430442810058594, -5.968924045562744, -9.736165046691895, -23.561582565307617, 11.33115005493164, 12.882631301879883, -1.1006762981414795, 3.694293260574341, -22.016067504882812, 6.959977149963379, -21.62723159790039, 14.554773330688477, -19.892253875732422, 25.244033813476562, -16.601016998291016, -3.196303367614746, 18.993202209472656, 0.776966392993927, -7.0037522315979, 23.41837501525879, 36.19516372680664, 5.962411880493164, 20.092479705810547, 7.566867828369141, 14.178168296813965, 2.8335206508636475, 35.997501373291016, -7.014703273773193, 7.584636211395264, 19.07137107849121, -10.415292739868164, -18.48637580871582, 30.517858505249023, -7.790762424468994, 25.7408504486084, -9.465010643005371, 23.452363967895508, 5.543956279754639, 19.952983856201172, 25.0566349029541, 19.381717681884766, -5.844422817230225], [9.928328514099121, 34.265708923339844, 6.604388236999512, -17.123334884643555, 19.546430587768555, -10.050494194030762, 22.453319549560547, 26.255586624145508, -2.6956965923309326, 26.77159309387207, 6.518558979034424, 18.2989501953125, -0.7032642960548401, -19.18138313293457, -5.759988307952881, 7.253559112548828, -4.570557594299316, 9.36768627166748, -6.286536693572998, 5.258233547210693, 3.731410026550293, -4.445409774780273, -24.324344635009766, 1.9523988962173462, -13.384740829467773, -42.96627426147461, -17.619400024414062, 31.630857467651367, 12.63369369506836, -14.084306716918945, 19.934438705444336, 9.364740371704102, 6.830480098724365, -6.15478515625, 6.175547122955322, -41.6297607421875, 6.405790328979492, -13.833414077758789, -7.230018615722656, -22.459440231323242, 14.060149192810059, 8.933104515075684, 8.0927152633667, 11.615208625793457, 1.6398996114730835, 2.1104462146759033, -24.703401565551758, -7.64796257019043, -24.762115478515625, -9.496058464050293, 10.151593208312988, -12.299090385437012, 5.797277450561523, 1.8294063806533813, 24.14168930053711, -5.932409286499023, 11.644808769226074, 6.325881004333496, 34.90974044799805, -31.071367263793945, 1.1349213123321533, 20.27732276916504, 44.416500091552734, 15.916705131530762], [-11.469014167785645, -31.452863693237305, 0.6084811091423035, -13.915643692016602, -0.3730144202709198, 5.991590976715088, -6.872159957885742, -9.564420700073242, -4.450136184692383, -18.047550201416016, 6.20054817199707, 7.53774356842041, 0.7268989086151123, -1.2271792888641357, 20.856475830078125, -13.79771900177002, 4.46751594543457, 29.375581741333008, 3.7249810695648193, -17.42792320251465, 33.81235885620117, 7.895700931549072, 17.50543975830078, 43.87015151977539, 6.401098251342773, -3.1938424110412598, -0.8260089159011841, -8.295808792114258, -2.5781760215759277, -6.985196113586426, 4.236070156097412, -18.280437469482422, -24.444866180419922, -14.106039047241211, 10.157967567443848, -4.299697399139404, 50.309146881103516, -0.978038489818573, 7.9745564460754395, -0.0269329734146595, -8.516456604003906, 26.008731842041016, 41.485206604003906, -8.968249320983887, 16.435054779052734, -23.292705535888672, -21.47252655029297, 6.543156623840332, -16.187604904174805, -4.4047064781188965, -11.658431053161621, 29.691816329956055, -9.051042556762695, -0.4838791787624359, 12.092289924621582, -6.055224895477295, -10.39046573638916, -10.215801239013672, -13.000044822692871, 42.53262710571289, 2.846081495285034, -17.31332778930664, 5.662169933319092, 12.710744857788086], [-1.3528610467910767, -23.99240493774414, 19.176607131958008, -5.300175666809082, -16.063220977783203, -22.941247940063477, -12.044818878173828, -6.662749290466309, -14.091572761535645, 5.113495349884033, -10.733323097229004, -17.993520736694336, 4.681720733642578, 12.612786293029785, -5.891657829284668, -11.08237075805664, 9.008485794067383, 1.6688255071640015, -33.03554153442383, 4.564029216766357, -16.524311065673828, -12.013229370117188, 2.9034690856933594, -21.933298110961914, -21.933639526367188, 1.7050186395645142, 21.551944732666016, -7.450181007385254, 17.612947463989258, 1.833187460899353, 3.582890510559082, 11.79483413696289, 6.33186149597168, -8.707850456237793, -8.213396072387695, 14.765694618225098, 5.020036697387695, -0.5577619671821594, -17.162824630737305, -6.15684700012207, -13.98466682434082, 2.886782169342041, 18.21501350402832, -6.6851677894592285, -28.94825553894043, 21.498937606811523, -1.2582628726959229, 17.245500564575195, 27.846223831176758, 29.044466018676758, 1.9451920986175537, -15.852357864379883, -12.556784629821777, 4.996825218200684, 8.802657127380371, 2.389983654022217, -37.44746398925781, 9.073753356933594, 9.039144515991211, -8.506067276000977, -6.982132911682129, 4.037890911102295, -13.368606567382812, 13.649229049682617], [-25.12384796142578, 2.7013487815856934, -4.281510353088379, 12.798523902893066, -8.829834938049316, -10.39289379119873, 6.053959846496582, -4.927481174468994, 19.415618896484375, 24.90437889099121, -1.171980619430542, -14.441524505615234, -6.324087619781494, 18.839412689208984, 11.743677139282227, 16.165441513061523, 18.549673080444336, -14.56606388092041, -28.34758758544922, 7.923142910003662, -30.657272338867188, 1.8374714851379395, -3.0846431255340576, -24.773635864257812, 20.551467895507812, -23.86723518371582, -30.336830139160156, 15.17601203918457, -5.716230869293213, 21.282739639282227, 14.276867866516113, -17.123146057128906, 22.867961883544922, 20.0963134765625, -26.41639518737793, 4.882383346557617, 19.491056442260742, -11.857336044311523, 2.495647430419922, -0.5378636121749878, -7.026507377624512, 8.987661361694336, -16.03173828125, 22.96160888671875, -9.21705436706543, -9.628255844116211, 20.7552547454834, -37.12024688720703, -2.01694917678833, -27.7241153717041, -0.051164623349905014, 3.6051902770996094, -37.762752532958984, 0.4633232057094574, 36.66667556762695, 21.251501083374023, -18.585586547851562, 28.148019790649414, -3.863760232925415, 2.9943320751190186, -1.266772747039795, 53.6451416015625, 1.221551537513733, 17.5507755279541], [-11.60795783996582, 2.0222134590148926, -6.300370693206787, -3.804617166519165, -0.9162792563438416, -11.717121124267578, 9.915300369262695, -53.69618225097656, 10.072291374206543, -9.30739688873291, 4.344790458679199, 6.692760467529297, -0.7201270461082458, -1.4486758708953857, 16.073335647583008, -21.518980026245117, 19.062456130981445, -1.4026710987091064, -14.273496627807617, 18.02347755432129, -2.705185651779175, 15.979802131652832, -19.846153259277344, 2.77474045753479, -8.489204406738281, 1.6605936288833618, 14.204880714416504, -29.7113094329834, -25.714719772338867, 19.73969268798828, 24.861560821533203, -4.409195899963379, 3.6804606914520264, -29.235029220581055, 7.553187370300293, -1.0170408487319946, 19.727333068847656, -1.3013173341751099, 4.796154499053955, 18.748598098754883, 20.077285766601562, 2.168605089187622, 2.6708693504333496, 5.268583297729492, 15.454789161682129, -13.785445213317871, 17.896989822387695, 4.12001371383667, -1.8663549423217773, 7.6298017501831055, 20.62217903137207, 4.947821617126465, 30.831607818603516, 4.512454032897949, 10.871994018554688, -32.82441711425781, -4.370659828186035, -6.190779209136963, 43.328086853027344, 11.927229881286621, -21.637096405029297, -9.593188285827637, -3.196404457092285, -24.94377899169922], [-9.084585189819336, -10.377673149108887, -12.996238708496094, 2.1912426948547363, 13.564474105834961, 21.34878921508789, -3.149245023727417, 7.184307098388672, 11.486124992370605, -32.814353942871094, -16.212554931640625, -10.540705680847168, 4.69520902633667, -6.603089332580566, 13.836857795715332, -5.274004936218262, 6.707372188568115, -26.490455627441406, 17.84419059753418, -8.378891944885254, -12.027026176452637, 10.32616901397705, -10.515172958374023, -36.44148635864258, 11.265932083129883, -13.030984878540039, -11.507269859313965, 4.617130279541016, 9.595948219299316, 13.896153450012207, -7.899428844451904, -6.829069137573242, -2.8184025287628174, -8.608402252197266, 7.049156665802002, -2.851369857788086, 5.6634721755981445, 12.294404029846191, 13.44405746459961, 4.979076862335205, -20.38593101501465, -19.109638214111328, 10.365056991577148, 29.121355056762695, -14.163020133972168, 14.381765365600586, -26.753150939941406, 9.102715492248535, 11.924924850463867, -19.928983688354492, -14.178658485412598, 1.1369789838790894, 9.27116870880127, -0.141009122133255, -2.91276216506958, -9.193926811218262, -0.0724993497133255, 3.8027403354644775, 13.661355972290039, -6.866888999938965, 6.3170013427734375, -20.80303955078125, 4.36466646194458, 43.15388488769531], [6.76254415512085, -2.781254768371582, -3.8450326919555664, -1.6316596269607544, -0.1360924243927002, 5.362605571746826, 13.05473804473877, 4.711428165435791, -17.251100540161133, -2.2105016708374023, 4.927746295928955, 21.797679901123047, 0.426240473985672, -38.46120834350586, 3.6508021354675293, 12.600029945373535, -30.09345054626465, 29.185575485229492, -1.9746057987213135, -0.8314493894577026, 5.997042179107666, 36.089515686035156, 2.3025338649749756, -35.434349060058594, -14.338492393493652, 40.352237701416016, 37.53750991821289, -9.698243141174316, -7.711973667144775, 17.952953338623047, 7.159918308258057, -0.8446266651153564, 0.2891235649585724, 10.253619194030762, -14.203852653503418, -12.479270935058594, -1.5507687330245972, -0.26711440086364746, -1.3689439296722412, -2.4974260330200195, 9.551229476928711, -28.345081329345703, -12.922693252563477, -11.098554611206055, 14.140558242797852, -0.7900825142860413, 2.530900001525879, -17.694284439086914, 0.5786367654800415, 10.258084297180176, -0.6476998329162598, 16.37462043762207, 12.040656089782715, -10.581771850585938, 1.7581218481063843, 13.022940635681152, 16.24371910095215, 17.364015579223633, -2.642848253250122, 5.199827671051025, -18.71638298034668, 8.01744556427002, -9.774855613708496, 12.69908618927002], [-8.942014694213867, -4.732399940490723, -12.320415496826172, 7.7093892097473145, 14.489166259765625, -16.37642478942871, 14.405436515808105, 18.436677932739258, -31.201557159423828, 0.8885860443115234, -11.102407455444336, -3.510132074356079, -6.9086408615112305, -18.784931182861328, 4.123281478881836, 7.361451625823975, -0.6855898499488831, -8.109635353088379, -13.879447937011719, 7.770161151885986, -2.5133750438690186, -6.243206024169922, 7.18515157699585, 4.805764198303223, -5.263129234313965, -1.3878082036972046, 1.4272361993789673, -9.987120628356934, -7.704813003540039, 4.661031246185303, 8.76449966430664, -7.33494234085083, 0.5831781625747681, -19.194856643676758, 19.05907440185547, 1.6962767839431763, 10.100643157958984, 27.959917068481445, -6.7144269943237305, 9.428455352783203, 19.93086814880371, -15.651938438415527, -9.550207138061523, 21.09062385559082, -16.83279037475586, 12.011027336120605, 3.7416436672210693, 15.94555950164795, 12.765219688415527, -7.397515773773193, -0.7928950190544128, -27.739412307739258, -1.300880789756775, -34.71595764160156, 7.209561824798584, 15.81903076171875, 27.21565818786621, 12.824268341064453, 12.802087783813477, -2.7052197456359863, 15.031087875366211, 12.959888458251953, -9.901731491088867, 16.686111450195312], [8.37626838684082, 8.338312149047852, -9.061878204345703, 23.540569305419922, -5.420692443847656, 3.861593008041382, 16.59701156616211, -21.83049201965332, -7.919593811035156, 10.68172550201416, 23.10927391052246, 5.6838603019714355, 2.4515693187713623, 3.8148794174194336, -6.967833995819092, 6.393874645233154, -32.0417594909668, 10.870101928710938, -1.9058153629302979, -6.900035381317139, -1.3516181707382202, -22.086841583251953, -0.7384091019630432, 16.37444305419922, 17.27273178100586, -15.220075607299805, -4.887901306152344, 20.394498825073242, -21.873167037963867, 18.83751106262207, -13.071331977844238, 14.739879608154297, 12.045844078063965, -21.76512908935547, 28.708946228027344, 10.175392150878906, -5.902736663818359, 24.437829971313477, 8.40031909942627, 15.468474388122559, 26.839468002319336, -3.202434778213501, 2.9773342609405518, 12.986360549926758, -15.675783157348633, -6.575983047485352, 18.118545532226562, 26.739194869995117, -7.597273826599121, -18.65894889831543, 8.90954303741455, 20.060701370239258, 35.3370475769043, 20.177589416503906, -19.02962303161621, 11.608569145202637, 12.520648956298828, 13.844344139099121, -3.2806336879730225, -8.745155334472656, -21.54808235168457, -23.015966415405273, 9.056975364685059, 15.604629516601562], [0.9810013771057129, 3.2660460472106934, 12.683609962463379, 4.678020477294922, 18.541582107543945, 22.008947372436523, -6.991519927978516, -0.9730579257011414, 31.721620559692383, 12.59582233428955, -23.817611694335938, -26.348552703857422, 9.051726341247559, -5.757134437561035, 24.944215774536133, 3.4864249229431152, 14.056953430175781, 14.13342571258545, -3.45721435546875, -17.509952545166016, 26.784608840942383, -13.921136856079102, 7.865521430969238, 3.7643048763275146, -11.413229942321777, 1.1343932151794434, 14.576563835144043, 2.4227304458618164, 4.150710582733154, -12.313117980957031, 10.871444702148438, -1.1771247386932373, -14.092486381530762, 16.910993576049805, -16.75677490234375, -0.6000651717185974, -14.1170015335083, -27.122600555419922, 7.794625282287598, -10.199419021606445, -5.181595325469971, 14.889764785766602, -13.384137153625488, 32.160133361816406, -15.980178833007812, 1.4278745651245117, 4.408203601837158, -9.675374984741211, 8.819911003112793, -3.9605350494384766, -0.0784696415066719, 4.412895679473877, -41.76301574707031, -11.433809280395508, 9.034608840942383, 5.539031982421875, -8.996615409851074, -34.489036560058594, 11.501938819885254, -1.5331635475158691, 34.41808319091797, -21.008092880249023, -7.296178817749023, 32.13307189941406], [24.13447380065918, 4.366660118103027, 1.6217623949050903, 19.071853637695312, 12.059223175048828, -19.438282012939453, 7.542807579040527, -6.188159465789795, -11.478105545043945, -47.325782775878906, -6.275632381439209, -3.3009772300720215, -14.814587593078613, 3.322908878326416, -7.252013206481934, -5.622400283813477, 2.5354740619659424, -29.03126335144043, -6.477363109588623, -15.197944641113281, 10.983377456665039, -17.41737937927246, -2.4515554904937744, -37.8299674987793, -5.1132330894470215, -7.505616664886475, -2.526707172393799, -41.10329818725586, 15.242817878723145, 0.3135048449039459, -3.2603280544281006, 19.75100326538086, 13.644619941711426, -11.493592262268066, -15.08167552947998, 3.2755396366119385, 11.306143760681152, 8.780597686767578, 4.516034126281738, 2.6741645336151123, 12.883102416992188, -6.958569049835205, -7.536250114440918, -24.571765899658203, -60.78044509887695, 1.8991451263427734, -3.496650218963623, 0.21223752200603485, -5.553750991821289, -3.6605169773101807, 5.665097713470459, -5.461373805999756, 20.534435272216797, -9.575225830078125, -1.4890252351760864, -29.73526382446289, 15.370197296142578, -2.831928253173828, -1.953478455543518, -17.739755630493164, -24.857160568237305, 25.5277156829834, -7.885343551635742, -7.576632976531982], [22.58226776123047, 20.64647102355957, -6.693870544433594, 7.260016918182373, -23.94206428527832, -19.01889419555664, 13.71628475189209, -10.32376766204834, 6.171940326690674, 1.4298291206359863, -17.960407257080078, -5.267330169677734, 11.458053588867188, -8.76070785522461, -45.141357421875, 4.789236068725586, 10.963258743286133, -37.28082275390625, 4.089985370635986, 13.805222511291504, -6.227601528167725, 1.8178974390029907, -3.1038036346435547, -15.41564655303955, 4.452303886413574, -0.013792385347187519, 39.5973014831543, -18.74273681640625, -6.150779724121094, -21.515474319458008, -26.669065475463867, -32.878578186035156, -3.9211230278015137, 15.301671028137207, 3.4371376037597656, -48.701595306396484, 16.565567016601562, 5.297542095184326, -9.566069602966309, 33.68672561645508, -1.5383625030517578, 12.234883308410645, -30.508411407470703, -37.706031799316406, -25.327102661132812, 2.488302707672119, 1.742142677307129, -7.0736541748046875, -2.9395153522491455, -7.800380229949951, -34.468441009521484, -11.68326473236084, 42.66152572631836, 1.5069700479507446, -8.476905822753906, -16.370595932006836, 33.491355895996094, -2.2073304653167725, 25.55336570739746, -13.556352615356445, 8.14154052734375, 21.090473175048828, 8.029366493225098, -15.881429672241211], [1.5620156526565552, 11.159384727478027, 9.083797454833984, -19.955188751220703, -9.031951904296875, 24.052734375, 5.470896244049072, 19.74053192138672, 16.937820434570312, -9.173739433288574, 7.667062759399414, -3.2787280082702637, 2.881328821182251, 6.0637335777282715, 16.354183197021484, -14.845928192138672, 8.848648071289062, -9.863414764404297, -2.300238847732544, -0.01179901696741581, -9.209977149963379, 20.556655883789062, -12.022207260131836, 9.579819679260254, -19.796878814697266, 5.245719909667969, -2.083108425140381, -9.49225902557373, 31.92339515686035, 8.525457382202148, 10.824217796325684, -21.352834701538086, 1.3988325595855713, 20.202011108398438, 11.725780487060547, 15.4653902053833, 3.450172185897827, 7.087327003479004, -14.092755317687988, -9.787425994873047, -7.158975601196289, 24.340986251831055, -4.475205898284912, -2.9241714477539062, -4.480188846588135, -10.942723274230957, -14.167550086975098, -2.292948007583618, 9.436054229736328, 1.4495739936828613, -0.9691951870918274, -13.99980640411377, -38.39289855957031, -5.405377388000488, 6.247329235076904, -7.624362945556641, -20.66521644592285, -12.801529884338379, -32.24908447265625, 14.17646312713623, 2.3041789531707764, -5.039333820343018, -12.791646003723145, -31.689836502075195], [2.3167197704315186, 15.509062767028809, -2.4121322631835938, -11.600323677062988, 6.570762634277344, -9.564353942871094, -17.24925994873047, -5.614348888397217, -10.397637367248535, 35.27311706542969, -21.10938262939453, 14.785967826843262, -3.8698337078094482, 0.9206454753875732, -33.19879150390625, -5.050691604614258, 0.26604974269866943, -12.339133262634277, -13.937651634216309, -15.729522705078125, 13.737447738647461, -1.245147943496704, -27.91146469116211, 4.363400936126709, -14.35529613494873, -15.976165771484375, -14.005025863647461, 27.031761169433594, -4.767849922180176, -18.918533325195312, -7.593206882476807, -4.440247535705566, -5.893235683441162, -17.717159271240234, 0.8662600517272949, -19.82449722290039, -4.394990921020508, -24.43488121032715, 15.88402271270752, -4.194245338439941, 14.86042594909668, -1.3483115434646606, 7.178762435913086, -5.521502494812012, -13.046595573425293, 7.071079730987549, -11.512516975402832, -8.653573989868164, -0.29588353633880615, -23.792455673217773, -16.926841735839844, -5.488443851470947, -25.84857940673828, -21.856666564941406, 4.754857063293457, 9.34071159362793, 30.823280334472656, 15.461685180664062, -2.042604446411133, -24.65045928955078, 6.184938430786133, 29.95695686340332, -0.07488595694303513, 18.10282325744629]], [[1.0823962688446045, -0.9209453463554382, -8.111407279968262, 7.573661804199219, -24.603727340698242, -12.242348670959473, -24.801372528076172, -15.208906173706055, -42.56611251831055, 12.651521682739258, 20.109657287597656, 12.053046226501465, 16.137149810791016, -0.7300099730491638, -4.101884841918945, 5.903708457946777, 8.732433319091797, 18.358600616455078, -15.98081111907959, 0.3361494839191437, 11.416942596435547, -8.750858306884766, 18.804231643676758, 8.525152206420898, -0.5157806873321533, -3.358644723892212, 7.483049392700195, -25.54414939880371, 21.714824676513672, 5.470115661621094, 21.492313385009766, 28.807092666625977, -8.9396333694458, -31.899049758911133, 0.5521836280822754, -23.330303192138672, -5.708345890045166, -5.954901695251465, -25.89058494567871, -5.376795768737793, -3.569629430770874, -0.4328990876674652, 24.959096908569336, -6.183009624481201, 10.002689361572266, -15.343277931213379, 10.48940658569336, -18.4880313873291, 5.761169910430908, 2.374238967895508, -14.124767303466797, -3.6121928691864014, 3.0935308933258057, -8.160527229309082, -14.56266975402832, 6.414099216461182, -34.20128631591797, -21.791011810302734, 2.7404887676239014, 20.713647842407227, -22.976055145263672, -35.5228157043457, -4.562042236328125, -25.46210479736328], [17.263198852539062, 7.992534637451172, 11.500667572021484, 16.169506072998047, 2.750265598297119, -6.95599889755249, 9.873541831970215, 20.91716766357422, -23.334001541137695, -9.422274589538574, -6.651810169219971, 16.60988998413086, 19.22895050048828, -16.331064224243164, -12.46589469909668, -15.35399055480957, 7.0100297927856445, -2.074241876602173, -10.573799133300781, -9.52946949005127, 12.667048454284668, 17.378114700317383, -17.856164932250977, -13.345549583435059, 21.929821014404297, 0.07283886522054672, -2.4575319290161133, -14.447628021240234, 2.3525569438934326, -0.024517839774489403, -14.784385681152344, -4.092411041259766, -18.460485458374023, -25.112817764282227, 13.77297592163086, 29.907625198364258, 5.018893718719482, -5.628746509552002, -13.27725601196289, -3.7501327991485596, -39.84703063964844, 1.1713451147079468, 23.39361572265625, -1.6278340816497803, -2.4900858402252197, 28.24968719482422, -7.48933744430542, -14.826745986938477, 3.290818452835083, 10.455853462219238, -10.5833158493042, 32.124305725097656, 6.717747688293457, 0.7421615123748779, 14.917532920837402, -13.933476448059082, -13.165534973144531, -23.57075309753418, 28.16620445251465, 8.814255714416504, -0.8978299498558044, 8.263873100280762, -17.344383239746094, 17.251087188720703], [10.295997619628906, -2.817721366882324, 30.90713882446289, -23.75425148010254, 3.342190742492676, 9.438995361328125, 8.565174102783203, -9.187832832336426, -16.646623611450195, 6.420165061950684, -7.8056206703186035, -22.94139862060547, -11.947885513305664, -18.70368194580078, -3.7764413356781006, 26.34188461303711, -4.06160831451416, 10.333683013916016, 18.304372787475586, -7.881114959716797, -20.249103546142578, -10.402289390563965, -36.36091613769531, 4.881932258605957, 3.6882009506225586, 18.77345848083496, -4.3859543800354, -7.8139824867248535, 0.580860435962677, -5.252665042877197, -6.370689868927002, -24.420446395874023, -8.00131893157959, 23.54671287536621, -17.569116592407227, 5.073190212249756, -16.45644187927246, 17.514629364013672, 8.99405288696289, 15.118219375610352, 17.277437210083008, -46.33512878417969, 1.4337196350097656, 7.256361961364746, -0.928457498550415, 5.795045852661133, 2.3409476280212402, 1.8863139152526855, -0.6836664080619812, -13.79403305053711, -24.027713775634766, -7.691227436065674, -25.628925323486328, 23.159116744995117, 13.269431114196777, -9.278215408325195, -7.1725077629089355, 18.32964515686035, -31.929391860961914, 21.096818923950195, 1.5832808017730713, 29.20431137084961, 34.42716598510742, 6.922695159912109], [-4.556467533111572, -26.99085807800293, 19.84103012084961, 21.147506713867188, -3.865741729736328, 9.140708923339844, -17.75111961364746, -13.75736141204834, -6.155492305755615, 5.652841091156006, -17.656084060668945, 22.018939971923828, -3.405379295349121, 4.846787452697754, -23.171171188354492, 5.077225208282471, 4.475696086883545, -16.494121551513672, -4.527845859527588, 25.8055419921875, 2.5959928035736084, 4.258411407470703, -15.851653099060059, 4.386349678039551, -22.735044479370117, -10.21914005279541, 20.41287612915039, 3.2786076068878174, 17.312204360961914, -7.364346504211426, 11.05441951751709, 3.8934264183044434, 4.868373394012451, -17.170412063598633, -14.806750297546387, 9.797094345092773, 12.32679557800293, 6.1957106590271, -5.260565280914307, -24.310792922973633, -1.6915403604507446, 8.61268424987793, 5.863961696624756, 13.954533576965332, 2.2260582447052, -17.679046630859375, -19.587745666503906, -30.277021408081055, -16.5411376953125, 16.56713104248047, 48.28514862060547, 20.756877899169922, 32.973899841308594, -26.85900115966797, -19.576841354370117, -0.7377158999443054, -25.41338348388672, 11.281392097473145, 5.194639205932617, 3.5968055725097656, -11.237171173095703, 24.743911743164062, 3.3365421295166016, -15.927250862121582], [-13.937596321105957, -4.857949733734131, -6.4734272956848145, 10.207292556762695, -26.442930221557617, 1.6599377393722534, 5.113079071044922, -24.762435913085938, -3.493537187576294, -19.741151809692383, 21.536880493164062, -1.791683554649353, 7.587210655212402, -19.295310974121094, 11.348936080932617, 22.332067489624023, 5.821645736694336, 0.465021014213562, 1.725325584411621, 10.268274307250977, -2.341845989227295, -5.043677806854248, -10.127138137817383, -34.823909759521484, 27.245023727416992, -6.04896354675293, -2.35184907913208, 11.112011909484863, -0.9125615954399109, 37.26082229614258, 6.022442817687988, -0.6008933186531067, -11.47236442565918, 10.565847396850586, -17.554826736450195, 17.129302978515625, -13.574346542358398, -5.623424530029297, -33.53657531738281, -6.265216827392578, -5.385651588439941, -3.518364667892456, 14.537130355834961, -16.768156051635742, 10.971781730651855, 8.115490913391113, -26.31452178955078, -9.837088584899902, 16.64808464050293, -23.5675106048584, -25.55065155029297, -21.07225227355957, -10.230621337890625, -5.6975531578063965, -36.059051513671875, 15.617301940917969, 6.722586631774902, 18.422000885009766, 32.870548248291016, -46.528167724609375, -0.41349664330482483, 10.715415000915527, -9.273204803466797, 20.422231674194336], [-12.316620826721191, -30.556310653686523, -8.122185707092285, -16.123205184936523, 7.93549919128418, -20.911758422851562, 13.10248851776123, -33.20812225341797, 20.001880645751953, -6.542969226837158, 5.928610801696777, -5.219657897949219, -5.478733539581299, 0.724531888961792, -13.934351921081543, -14.872944831848145, 8.109244346618652, -11.430105209350586, 25.43741226196289, -10.331424713134766, -18.187484741210938, -0.4321008324623108, -23.790117263793945, -4.827515125274658, 8.15401554107666, -1.1798980236053467, 2.098984479904175, 20.194263458251953, 26.915008544921875, 23.890520095825195, 4.420026779174805, -5.00670051574707, 6.364076614379883, 18.38178062438965, -32.28237533569336, -8.360252380371094, 0.7761015892028809, -11.506136894226074, 9.158635139465332, -29.852663040161133, 10.20940113067627, -9.526455879211426, 4.414129734039307, -17.7401180267334, 3.2065467834472656, 4.251303672790527, -19.147457122802734, 8.448598861694336, 4.4423370361328125, -23.582857131958008, 4.945403575897217, -13.599018096923828, -10.893081665039062, 11.655752182006836, 12.363513946533203, 13.92695140838623, 17.060733795166016, -3.8508031368255615, -2.872142791748047, 8.498936653137207, 14.927986145019531, 41.578582763671875, -1.1144400835037231, -35.1314697265625], [25.99935531616211, -0.5059506893157959, -2.3668081760406494, -2.7395968437194824, 0.5783972144126892, 2.1322500705718994, -23.802322387695312, 10.260709762573242, 4.609841823577881, 29.172513961791992, -19.6451473236084, -5.843303680419922, -25.805700302124023, 33.1573371887207, -26.14825439453125, -7.561031341552734, -26.692140579223633, -6.736162185668945, -20.81534194946289, 4.439672470092773, -25.72248077392578, 28.11747169494629, 6.057402610778809, -11.863211631774902, -26.56281089782715, -4.638823509216309, 1.5174791812896729, -17.547832489013672, 8.098092079162598, 0.5084343552589417, 9.667881965637207, 6.072188854217529, 12.305173873901367, 7.510702133178711, -20.61264419555664, -3.7489399909973145, -2.887636184692383, 21.034364700317383, -6.279402256011963, 20.438823699951172, 2.869715929031372, -23.96925163269043, 2.8257341384887695, 30.74252700805664, -2.9830033779144287, -24.583595275878906, -6.860712051391602, 22.939199447631836, 14.87152099609375, -25.657434463500977, -17.291913986206055, -5.900234699249268, -6.55970573425293, 7.771274089813232, 8.299897193908691, -22.490991592407227, -13.856616973876953, -0.8408192992210388, -12.468338012695312, 16.042526245117188, 16.937305450439453, 10.931840896606445, -11.208454132080078, 8.194643020629883], [11.039325714111328, 32.99739074707031, 16.805004119873047, 4.2100677490234375, 7.572106838226318, -4.5774760246276855, 1.974402904510498, -19.156414031982422, -30.00307846069336, 15.435060501098633, 32.95635986328125, 20.501773834228516, 14.310230255126953, 25.96979331970215, -13.19229793548584, 19.41208267211914, 6.617345809936523, -20.570905685424805, 6.1197333335876465, -7.961912155151367, 7.513483047485352, 30.699949264526367, 7.837080001831055, 24.612049102783203, -1.0767379999160767, -30.394731521606445, 6.268435001373291, 5.422493934631348, 0.7340261340141296, -21.672189712524414, 4.384517192840576, -4.705679893493652, -31.139612197875977, -10.495088577270508, -4.744211196899414, -3.9586827754974365, 3.8109681606292725, 19.05108642578125, 10.081727027893066, -1.3165801763534546, 4.235528469085693, -0.41625234484672546, -4.537938117980957, -9.919776916503906, 9.838890075683594, 0.6929721236228943, 19.993934631347656, 1.4667339324951172, -40.6077766418457, 10.327909469604492, 3.6205496788024902, -16.492156982421875, 6.190748691558838, -11.928351402282715, 0.9966068863868713, 9.937374114990234, -3.3511693477630615, -13.603665351867676, 7.4663519859313965, 33.52225112915039, -14.974030494689941, 7.074151515960693, 6.368043422698975, 15.032238960266113], [19.113855361938477, -34.50617980957031, -15.750428199768066, 7.120513439178467, 5.63867712020874, 2.8804521560668945, -20.724807739257812, -9.840524673461914, 27.670780181884766, 14.28350830078125, -5.598141193389893, 4.949200630187988, 8.679325103759766, -6.294482707977295, 28.35523223876953, -19.62937355041504, 3.789698362350464, 1.640895128250122, 26.06350326538086, -8.264459609985352, 14.469013214111328, 29.608562469482422, -22.803953170776367, 9.503299713134766, 20.2283935546875, -10.904077529907227, 11.536972045898438, 19.487451553344727, -7.056243419647217, -17.456443786621094, 0.8786527514457703, -16.443500518798828, -11.858110427856445, 7.109243392944336, 21.138635635375977, -5.3402628898620605, -22.286399841308594, -1.6031274795532227, -10.846299171447754, 19.587373733520508, -43.27473449707031, -1.2332642078399658, 16.237232208251953, -2.5869438648223877, 17.283029556274414, -12.486997604370117, -22.536436080932617, -10.621454238891602, 0.13050909340381622, 10.942816734313965, 27.35413360595703, -7.678491115570068, 8.75128173828125, 1.0887117385864258, -7.833884239196777, 11.824883460998535, -0.05565700680017471, 4.334444999694824, -14.936944961547852, -23.762205123901367, 32.245933532714844, -25.965024948120117, -30.205930709838867, -6.332988739013672], [-16.157995223999023, -11.398938179016113, 39.553955078125, 12.254761695861816, -8.000911712646484, -6.595043182373047, -0.1938181221485138, -0.4553467035293579, -1.276922583580017, 8.284605026245117, -11.839728355407715, 18.662118911743164, -5.957282543182373, -4.164279460906982, -34.72150421142578, -6.874524116516113, 14.353090286254883, -9.703120231628418, -10.53727912902832, -10.107277870178223, 10.062917709350586, -12.392094612121582, -7.713068962097168, 8.026082992553711, 13.728754997253418, -4.066769599914551, 7.810821056365967, 8.642848014831543, -27.1290340423584, 2.3304779529571533, -39.71985626220703, 6.508273124694824, 22.577852249145508, -1.5454667806625366, 5.240524768829346, -3.862492799758911, 18.959522247314453, -9.663256645202637, 3.560642957687378, 8.27226448059082, 21.296594619750977, -2.0975019931793213, -41.80661392211914, 1.873766541481018, 25.046932220458984, -7.465461730957031, 7.81739616394043, 5.453860282897949, 6.917757511138916, -5.395138263702393, -35.654685974121094, -9.547568321228027, -12.66676139831543, 6.230529308319092, 6.487555980682373, -40.629798889160156, 21.251684188842773, 11.009087562561035, 14.476021766662598, 2.3019840717315674, 6.3050432205200195, -26.74463653564453, -0.019119445234537125, 0.11889629811048508], [-3.6806044578552246, 7.02730131149292, -13.734848022460938, 22.26443099975586, -2.776087522506714, -27.249862670898438, 3.049999713897705, 14.807315826416016, 21.482194900512695, -2.8913605213165283, 18.48553466796875, -7.747060775756836, -3.2559642791748047, -12.069547653198242, 0.7056295275688171, -9.950654029846191, -2.7076380252838135, 15.29330825805664, -20.309682846069336, 3.0202744007110596, 8.218575477600098, -0.8742212653160095, 17.561922073364258, 0.5046364068984985, 9.73048210144043, 10.01824951171875, -2.4761154651641846, -0.943701982498169, 12.386749267578125, 11.093157768249512, -26.453067779541016, 32.17878723144531, 28.21364402770996, -2.8621227741241455, -2.409135341644287, -17.730667114257812, 0.7714038491249084, -31.95077133178711, 32.175052642822266, 1.7309428453445435, -28.446380615234375, 28.396413803100586, 26.523340225219727, 1.4123080968856812, 19.03850746154785, 36.66168975830078, -14.64176082611084, -10.527589797973633, -8.514884948730469, -13.340400695800781, 3.2222979068756104, 21.89339256286621, -15.152080535888672, -3.198193073272705, -10.616518020629883, -16.375001907348633, 34.3067626953125, 7.4964985847473145, 17.026103973388672, 2.5990052223205566, 23.717716217041016, -4.921835899353027, 12.024612426757812, 16.785045623779297], [-20.507551193237305, 9.785405158996582, 0.8245465755462646, -20.060932159423828, 12.924930572509766, 10.704924583435059, -2.1524581909179688, -26.69257926940918, 22.125276565551758, -15.310660362243652, -4.2120795249938965, 10.554880142211914, 6.202249526977539, 22.563823699951172, 18.903369903564453, -8.789017677307129, -6.352832317352295, -9.306249618530273, -1.0888698101043701, -13.907819747924805, -13.775413513183594, -32.01517105102539, 7.044986724853516, -21.62445831298828, 15.047813415527344, -25.213294982910156, 10.381969451904297, 2.3099026679992676, 17.13404083251953, -2.7773454189300537, -7.479650020599365, 10.22283935546875, -15.034720420837402, 37.803565979003906, -3.9582607746124268, 11.039304733276367, 28.199026107788086, -27.58148956298828, -7.845834732055664, 38.447265625, 15.192541122436523, 36.68844223022461, 5.3943023681640625, -6.821834564208984, -7.413546562194824, 8.500411987304688, -1.0782674551010132, 19.467939376831055, -1.906647801399231, -13.1029634475708, 32.59154510498047, 38.771583557128906, -8.556632995605469, -17.09197998046875, -4.141984939575195, -22.040361404418945, 9.367349624633789, -15.204366683959961, 6.247071266174316, -6.273948669433594, -6.8819684982299805, 19.09759521484375, 16.208967208862305, -0.169560506939888], [16.4157657623291, -26.31680679321289, 4.756625175476074, -1.8952786922454834, -14.046446800231934, -41.08641052246094, -9.01870059967041, -11.578373908996582, 7.218419075012207, 14.639431953430176, 4.1203179359436035, 7.341400623321533, 12.87380313873291, -8.204461097717285, 19.078433990478516, 8.27053165435791, -10.231785774230957, 29.46677589416504, 2.9252231121063232, -1.1725175380706787, 3.380765199661255, -12.003767013549805, -16.039958953857422, -2.52553653717041, -1.5942153930664062, 9.121582984924316, 15.774798393249512, 35.680641174316406, -28.58877182006836, 7.376336097717285, -13.813629150390625, -4.114068031311035, 12.852703094482422, 4.702990531921387, -3.5331814289093018, 14.614928245544434, 13.698173522949219, 19.29624366760254, 8.939545631408691, 25.833208084106445, -7.9842705726623535, 18.252527236938477, 8.291153907775879, 19.094390869140625, 18.626157760620117, 14.971048355102539, -16.0843448638916, 1.7767521142959595, 4.508502006530762, 11.002603530883789, 7.345213890075684, 1.2353118658065796, 16.293899536132812, -9.379708290100098, 3.0011565685272217, -19.294530868530273, 22.958568572998047, 13.087662696838379, 1.114130973815918, -4.9946770668029785, 13.06860637664795, 7.698983669281006, -15.215555191040039, -21.33295440673828], [-7.861827850341797, 0.976382315158844, 1.7723121643066406, -15.62575912475586, -12.23355770111084, -23.831905364990234, 33.543975830078125, -5.462595462799072, -19.14080810546875, 0.35160350799560547, 0.2049589306116104, -5.045961856842041, 8.451082229614258, 1.9621753692626953, 3.4797251224517822, 4.883331298828125, -10.185037612915039, -3.9738564491271973, -27.354969024658203, 20.45224952697754, 0.6410746574401855, 4.296690464019775, -3.0894241333007812, -1.5772290229797363, 9.236984252929688, 23.66557502746582, -32.15121841430664, 2.2634212970733643, -14.83506965637207, 18.066835403442383, -52.996944427490234, -7.364963054656982, 28.195838928222656, 15.065732955932617, 9.052923202514648, 36.611610412597656, 12.28896427154541, 1.941174030303955, 23.360431671142578, -3.457198143005371, 20.78911018371582, -45.723228454589844, 24.422687530517578, -8.770017623901367, 6.001819133758545, -13.014180183410645, 25.118932723999023, -5.679474353790283, 7.414848327636719, -12.591059684753418, -17.804340362548828, 10.744488716125488, 1.4686812162399292, 7.70033073425293, 27.367311477661133, 3.047208309173584, 10.340102195739746, -14.310258865356445, 3.098419427871704, 9.243289947509766, -0.015823185443878174, 16.466411590576172, 16.691143035888672, 7.5467071533203125], [-26.11768341064453, 0.525750994682312, -11.590225219726562, -3.0110599994659424, 8.77084732055664, 17.686731338500977, 9.646768569946289, -23.04593849182129, 4.248751640319824, -10.425093650817871, -4.468659400939941, -11.17431926727295, 10.788436889648438, -2.874952793121338, 6.6996564865112305, -4.024477005004883, 6.402796268463135, 9.70283317565918, 26.8099422454834, 24.73853874206543, 26.103759765625, 7.531187057495117, -10.988770484924316, 3.769054651260376, 8.815396308898926, -3.9371519088745117, 1.0722236633300781, -5.972926616668701, 29.606746673583984, -7.2826642990112305, 11.305797576904297, 10.079645156860352, -28.458147048950195, -16.41675567626953, -43.73222351074219, 21.280851364135742, -8.281961441040039, -16.580053329467773, -12.080354690551758, -40.7320671081543, -21.313034057617188, -30.41267204284668, -9.012428283691406, -6.43946647644043, 19.301536560058594, -6.364315509796143, -21.396541595458984, 7.77046537399292, -5.176749229431152, -5.703970432281494, -27.661169052124023, -21.01010513305664, 10.269061088562012, -13.850083351135254, 10.024924278259277, 7.034623146057129, -25.849071502685547, 0.9415135383605957, 30.861265182495117, 21.59279441833496, 13.886626243591309, 23.805994033813477, -21.00221061706543, -20.168054580688477], [-22.88551902770996, 17.898048400878906, -0.30203142762184143, -7.705362796783447, 29.57362937927246, -28.769926071166992, -2.451826333999634, -11.688340187072754, -0.46466541290283203, -9.21985149383545, -11.726014137268066, -6.528564929962158, 19.93090057373047, -26.95970344543457, -5.859536170959473, 0.043291378766298294, 3.640537738800049, -27.93285369873047, 0.8327257037162781, -5.6362528800964355, 6.643314361572266, 17.83769989013672, -16.800331115722656, -2.914505958557129, -17.705989837646484, 3.3922412395477295, 15.036784172058105, 8.404145240783691, -9.752140045166016, 32.08369445800781, -13.76672649383545, 13.152876853942871, 12.878389358520508, 11.50665283203125, 24.08819580078125, -4.826199054718018, 18.206684112548828, -20.548919677734375, 26.069652557373047, 3.1563546657562256, -18.142555236816406, -5.608818054199219, 9.30774974822998, -14.379597663879395, 5.674279689788818, 6.264595031738281, 18.552377700805664, 24.06360626220703, -25.651531219482422, 10.469239234924316, -45.28407669067383, -7.693897247314453, -7.464491367340088, 0.33206820487976074, -4.427269458770752, 23.19007110595703, -8.272130012512207, -10.127483367919922, 24.53440284729004, -14.721494674682617, 4.748648166656494, -18.903968811035156, -0.36810117959976196, 29.602399826049805]], [[-30.537946701049805, 14.171822547912598, 0.8885722756385803, 4.952645301818848, 7.857238292694092, -10.78307056427002, -3.749009847640991, -0.23219865560531616, 10.374481201171875, -15.953044891357422, -22.00322151184082, 10.950617790222168, -3.4322118759155273, 5.828746795654297, 3.402247190475464, -37.680145263671875, 5.315167427062988, 18.469846725463867, -23.084360122680664, 21.999608993530273, -1.9004864692687988, 4.185161590576172, 13.989490509033203, 3.25179123878479, -3.042588472366333, 1.0839240550994873, 15.384614944458008, 18.047937393188477, 27.379905700683594, 25.392301559448242, 5.968409061431885, -4.2780961990356445, 0.4756758213043213, 41.910499572753906, -19.98827362060547, 3.8953120708465576, 12.28936767578125, -0.20354419946670532, 16.686559677124023, 9.803548812866211, -12.405736923217773, -4.159571170806885, 8.998183250427246, -7.420887470245361, 3.87215256690979, 2.719923734664917, 6.667966365814209, -15.706543922424316, 18.011383056640625, 6.289006233215332, -3.1199309825897217, -14.950390815734863, -10.759803771972656, 21.060453414916992, -6.566054344177246, -3.6116373538970947, 19.46556854248047, 3.3060803413391113, 7.538868427276611, -32.48819351196289, 10.02817153930664, 10.298744201660156, -7.043445110321045, -2.3798086643218994], [-1.2531013488769531, 13.757975578308105, -3.068408489227295, -12.410768508911133, -6.539730072021484, 6.618359088897705, -15.883487701416016, -18.321680068969727, -8.229063987731934, 14.316274642944336, 9.191987991333008, -14.575780868530273, 16.365070343017578, -15.240961074829102, 6.4557976722717285, -21.167234420776367, -4.187924385070801, -31.103723526000977, -12.772892951965332, -8.9634428024292, -0.18685948848724365, -15.414249420166016, -22.74445343017578, -17.118268966674805, -2.0383803844451904, 1.9682049751281738, 19.14537239074707, -15.011434555053711, 9.118200302124023, -7.617225170135498, 0.9577634930610657, -1.435218095779419, -6.117568016052246, 16.981863021850586, -19.488155364990234, -18.90254783630371, 21.171443939208984, 11.423608779907227, -25.47624969482422, 10.793291091918945, -13.31601333618164, 18.010915756225586, 5.361177921295166, 1.3599629402160645, -12.51669692993164, 5.759969711303711, 13.655611991882324, 18.917192459106445, 4.067107200622559, 30.670705795288086, 0.024293355643749237, 4.96759033203125, 40.634727478027344, -0.31524747610092163, -7.464598655700684, 51.90779113769531, 5.074258327484131, 24.7740421295166, 2.189688205718994, 1.531637191772461, 9.042655944824219, 6.338520526885986, -14.167431831359863, 5.980842590332031], [-6.859749794006348, 19.71419334411621, -3.0332698822021484, -19.222124099731445, -9.969799995422363, -6.054981708526611, -11.576635360717773, -13.416850090026855, 5.493935585021973, 26.14645767211914, 24.215526580810547, -52.45893859863281, 6.142760753631592, 16.108497619628906, 13.325305938720703, 6.879189968109131, -14.512977600097656, -6.247201919555664, 3.0540692806243896, 6.0384697914123535, 8.638996124267578, -24.56525421142578, 2.9094583988189697, 69.03001403808594, 16.383129119873047, -5.750392913818359, 3.842689275741577, -7.233784198760986, -21.853233337402344, -15.427173614501953, 6.766027450561523, -28.034404754638672, 17.709138870239258, -2.660536289215088, -1.6333576440811157, 52.66990661621094, 10.390911102294922, -9.545238494873047, 4.299137115478516, -19.034521102905273, -18.093955993652344, 8.185990333557129, 17.051454544067383, 8.059661865234375, 3.0496413707733154, -15.70075511932373, -19.778793334960938, -2.5668249130249023, 4.333684921264648, -9.72734260559082, 28.34586524963379, 9.342728614807129, -18.372220993041992, 22.475988388061523, -24.16219139099121, 16.52543830871582, 24.084884643554688, -36.23242950439453, 5.77116060256958, 5.381793975830078, -9.280585289001465, 14.696329116821289, 6.700576305389404, -19.072906494140625], [-11.66405963897705, -31.142417907714844, 7.308699131011963, -21.888811111450195, 13.21119213104248, -10.532156944274902, -0.030638063326478004, -7.693054676055908, -8.613044738769531, 1.7559819221496582, 5.8985724449157715, 3.5815391540527344, 2.4871137142181396, -4.647036075592041, -5.874612331390381, 11.953191757202148, 5.492251873016357, -4.31086540222168, 14.332646369934082, 36.650447845458984, 0.51759272813797, -27.746896743774414, 13.092248916625977, 4.590921878814697, 39.83901596069336, 12.222369194030762, 7.6480488777160645, 41.51229476928711, 3.888057231903076, -13.059002876281738, 15.073429107666016, 39.80221939086914, -20.97184181213379, 1.1782866716384888, 5.780703067779541, -18.71840476989746, -33.657997131347656, 0.20020462572574615, 8.202892303466797, 26.366655349731445, 12.042415618896484, -9.426651954650879, -3.7387948036193848, -16.329500198364258, -0.35868582129478455, 17.892980575561523, -5.32722282409668, -9.292759895324707, -10.078513145446777, -0.5866361856460571, 3.852593183517456, -11.910239219665527, 3.8499855995178223, 23.333789825439453, 21.898101806640625, 13.711710929870605, 13.366579055786133, 14.29733657836914, -19.564008712768555, 30.947175979614258, 22.44529914855957, -11.349896430969238, 32.048377990722656, 22.79326820373535], [7.825921058654785, -7.184418678283691, 7.182369709014893, -10.0878324508667, 0.6065018773078918, -28.77910041809082, 12.153603553771973, 1.8769028186798096, 5.113675594329834, 19.873205184936523, -18.961313247680664, 4.653000831604004, 24.249860763549805, 19.501943588256836, 10.037633895874023, -3.916919231414795, 16.827367782592773, -11.248257637023926, -2.959530830383301, 22.200780868530273, -29.907272338867188, -2.911928653717041, 32.154117584228516, 9.775038719177246, 24.159648895263672, 9.43368148803711, -16.655208587646484, 12.122203826904297, 26.079938888549805, -21.160232543945312, -3.3507962226867676, -9.23523235321045, -14.284192085266113, 12.00238037109375, 26.377296447753906, 14.999251365661621, -45.188289642333984, -7.928936958312988, -6.44402551651001, -9.650223731994629, -17.332136154174805, 7.088660717010498, 6.036818981170654, -14.948271751403809, -32.225040435791016, 9.195213317871094, -7.005870819091797, -0.9125896096229553, -4.897337436676025, 24.797080993652344, -12.525413513183594, -15.181587219238281, -5.231850624084473, 8.884748458862305, -14.99398136138916, 22.464197158813477, 4.283780097961426, -3.339564323425293, -23.059003829956055, 18.794797897338867, 10.450597763061523, 21.102745056152344, 23.144302368164062, 9.111279487609863], [-34.64802932739258, -14.2393217086792, -14.388459205627441, -7.303767204284668, 2.3450357913970947, -25.976621627807617, 15.111661911010742, 9.494061470031738, 11.587433815002441, -10.169669151306152, -20.904802322387695, 14.104320526123047, -40.07575988769531, -31.205833435058594, 1.769684076309204, 16.492717742919922, 17.371599197387695, 19.75259017944336, -4.5637898445129395, -1.408111810684204, -0.16464291512966156, 12.325238227844238, -20.54663848876953, -3.154649496078491, 4.31235933303833, -4.256265640258789, 21.775279998779297, 8.877696990966797, 28.006622314453125, 26.55849838256836, -12.80708122253418, 7.957283973693848, -13.560300827026367, -8.67575454711914, -14.36256217956543, -17.060352325439453, 0.22959329187870026, 15.170555114746094, -30.579795837402344, 3.747378349304199, 9.468278884887695, -3.6235482692718506, -1.84165620803833, 13.301600456237793, -13.115303039550781, 4.285975456237793, 17.69850730895996, 3.8535563945770264, -13.002761840820312, -15.815512657165527, 0.37250688672065735, -12.922477722167969, -1.0269850492477417, -40.582950592041016, 13.57007122039795, 5.2814130783081055, 8.548530578613281, 9.898322105407715, -8.638033866882324, 15.198192596435547, -17.817171096801758, 1.0306411981582642, -2.085418462753296, -12.80162239074707], [22.507413864135742, -18.088529586791992, 2.737483501434326, -20.115415573120117, -17.11514663696289, -17.21343421936035, 18.45160675048828, -20.39201545715332, -18.448959350585938, 16.37763214111328, 17.617950439453125, 34.81413650512695, 14.732474327087402, -3.546367883682251, -1.6901909112930298, -16.42621612548828, -4.700536727905273, -8.947333335876465, -11.719632148742676, 9.116652488708496, -17.430034637451172, 21.895042419433594, -4.851099967956543, 2.743499994277954, -13.96700382232666, 3.9694948196411133, 15.634893417358398, -15.937433242797852, -21.731740951538086, -10.252821922302246, -12.133723258972168, 42.67330551147461, -15.333863258361816, -10.78303050994873, 9.656418800354004, 0.6839776635169983, 20.580415725708008, -19.40846824645996, 0.8382532596588135, -15.94664192199707, -2.536918878555298, 9.341447830200195, -14.282267570495605, -11.243666648864746, 21.16339111328125, 14.117142677307129, 14.175871849060059, -1.0675524473190308, -15.06932258605957, -2.416809558868408, 20.021547317504883, -9.442313194274902, -11.908512115478516, 2.914933919906616, -7.505590915679932, 4.838055610656738, -9.95160961151123, 24.922683715820312, 24.6445255279541, 26.020156860351562, -7.6655354499816895, -26.75971221923828, -1.3972225189208984, 8.282294273376465], [-10.18494987487793, -3.314354419708252, -6.691221714019775, 18.936838150024414, -6.32570743560791, 0.6570307612419128, 2.935032367706299, -10.717859268188477, 13.868375778198242, -19.71269416809082, -15.563583374023438, -5.442590236663818, -11.977862358093262, 6.534121513366699, -18.208412170410156, -15.114859580993652, -1.0276135206222534, 20.659683227539062, -29.140411376953125, 5.361481189727783, 1.8257575035095215, 13.702465057373047, 4.729753017425537, -3.601515531539917, 14.707010269165039, -1.4530494213104248, -9.71597957611084, -8.043593406677246, 5.186410903930664, 4.792105197906494, -6.9423604011535645, -1.4034761190414429, 0.35597434639930725, -10.737325668334961, 4.0393500328063965, -4.115789413452148, 5.13624382019043, -45.911766052246094, -15.269243240356445, 4.591898441314697, 13.698848724365234, 16.88229751586914, 16.5189208984375, -2.164700746536255, 3.1546919345855713, -15.787188529968262, -0.31939807534217834, 0.0895608514547348, -17.269187927246094, -25.923810958862305, 16.268726348876953, 8.937413215637207, -1.3349111080169678, 28.77159881591797, -4.3648786544799805, -6.795879364013672, 5.470827579498291, 24.640457153320312, 17.48299217224121, -13.965880393981934, -25.271060943603516, -6.736233711242676, 16.748294830322266, -28.53412628173828], [-3.638223648071289, 13.809136390686035, 9.048057556152344, 38.03594970703125, 11.53121566772461, -26.26811408996582, 17.535062789916992, -4.7635884284973145, 6.370616436004639, 15.723088264465332, -15.521882057189941, 5.391530990600586, 30.905916213989258, 2.910621166229248, 34.33022689819336, 23.10988426208496, -16.05695915222168, 0.4097057580947876, -19.351242065429688, -0.49062013626098633, -28.4588680267334, 11.063813209533691, -17.380126953125, 1.6216955184936523, 8.442038536071777, 10.96428394317627, -18.548389434814453, 1.7250487804412842, 0.9205849170684814, -0.8848181962966919, 15.583535194396973, -10.385135650634766, -21.270771026611328, -30.903615951538086, -5.187083721160889, -7.579898357391357, 1.3960081338882446, 2.1218435764312744, 6.94334077835083, 11.602248191833496, -8.128360748291016, 7.420230388641357, 15.630724906921387, 8.323163032531738, 0.40986478328704834, 6.511621475219727, -9.855631828308105, 26.48068618774414, -12.385159492492676, -33.283836364746094, -7.677401542663574, 15.031811714172363, -37.927738189697266, -41.413658142089844, -2.3306849002838135, 1.267100214958191, 33.346885681152344, -13.828893661499023, -27.196231842041016, 23.388559341430664, -8.926203727722168, -9.09542465209961, 27.791271209716797, -14.066917419433594], [-48.234092712402344, 1.5280643701553345, 2.942690849304199, 26.650676727294922, -1.231397032737732, -20.5416202545166, -30.072315216064453, 50.46287536621094, 9.331588745117188, -11.797900199890137, 11.755465507507324, -12.215899467468262, -25.40610694885254, 1.3408187627792358, 4.136343955993652, -10.23214340209961, 14.596036911010742, 22.420394897460938, 17.487449645996094, 17.367403030395508, -3.9330642223358154, 9.455408096313477, 2.871182918548584, -34.784481048583984, -2.1139895915985107, -22.004173278808594, -11.664923667907715, -17.77223777770996, -3.7371878623962402, 14.510417938232422, 18.796428680419922, -7.681099891662598, 36.32157897949219, 7.999391555786133, -1.261175513267517, -3.954306125640869, -2.4216363430023193, -13.904398918151855, -2.266738176345825, 14.810577392578125, -27.873592376708984, 0.25914645195007324, -8.895094871520996, -4.158468246459961, -18.41763687133789, 9.972082138061523, 1.7255747318267822, 1.8498300313949585, -15.384251594543457, 4.473114013671875, -26.657386779785156, -5.8377766609191895, -4.954051494598389, 4.750709533691406, 9.142081260681152, 25.96954917907715, 10.340654373168945, -9.351837158203125, 6.875016689300537, -20.5566463470459, 27.192237854003906, 16.997957229614258, -21.60036849975586, -16.136194229125977], [3.025141477584839, -17.323070526123047, 16.494054794311523, 3.6497747898101807, -5.381176948547363, -20.979724884033203, -7.413243770599365, -9.978363990783691, -16.228227615356445, 19.620296478271484, 1.8159735202789307, -4.621159553527832, -8.927249908447266, -42.90034866333008, 2.7756330966949463, -14.31511402130127, 1.6125177145004272, -13.425511360168457, 10.297982215881348, -2.0900204181671143, 13.32534408569336, -4.321078777313232, -33.848960876464844, -4.880673885345459, 3.8562026023864746, -13.652904510498047, 35.30495834350586, -12.411666870117188, -39.45695877075195, -6.208714008331299, -0.7001461386680603, 31.929244995117188, 2.761404514312744, 3.1409056186676025, -21.531620025634766, 6.867806434631348, 12.304705619812012, 4.650954246520996, -41.65707778930664, -14.91152286529541, -11.407052993774414, -25.872821807861328, -8.246752738952637, 16.60556411743164, 13.263858795166016, -8.898597717285156, -5.911494731903076, -8.865311622619629, -0.4457367956638336, 0.8946051597595215, 6.384071350097656, -16.607622146606445, 25.550914764404297, 2.4991135597229004, 12.197250366210938, 1.8776123523712158, 19.845640182495117, -15.663104057312012, 13.357357025146484, -20.727819442749023, 17.075944900512695, 0.6080093383789062, 1.0756442546844482, 9.705754280090332], [21.880943298339844, 15.958742141723633, 2.617094039916992, 13.291826248168945, 28.000825881958008, 4.851689338684082, -9.590782165527344, 15.171730995178223, 0.7378115057945251, 15.826827049255371, -21.143301010131836, -11.658312797546387, 7.203742504119873, 9.20815372467041, 35.36041259765625, 22.9121036529541, -31.51487159729004, -10.908681869506836, -4.434670925140381, 8.166088104248047, 1.0307940244674683, -12.279067039489746, -0.7229659557342529, -5.353289604187012, 2.7092971801757812, -13.365415573120117, -6.198155879974365, -17.830289840698242, -10.439628601074219, -16.4383544921875, -23.56665802001953, -27.506196975708008, 8.52456283569336, -0.45884349942207336, -9.624502182006836, 2.254149913787842, -5.715649604797363, -20.210947036743164, 0.7047569155693054, 5.971299648284912, 4.015720367431641, 0.3915262222290039, 17.0921630859375, 12.799972534179688, 12.44434642791748, -0.6151685118675232, 8.689359664916992, 19.207048416137695, 12.891830444335938, -3.4233577251434326, -9.361876487731934, -1.0147502422332764, -13.358723640441895, -3.6907360553741455, -7.74324893951416, 1.9006232023239136, -1.1901352405548096, -58.69981384277344, -30.425058364868164, 9.006200790405273, 11.625197410583496, -2.3720481395721436, -24.724885940551758, 4.941922664642334], [13.31301212310791, -24.323678970336914, 11.3589506149292, -10.330766677856445, -10.657500267028809, 37.27228927612305, 29.9511661529541, -4.791407585144043, -18.153226852416992, -8.57545280456543, -14.097238540649414, -1.6185684204101562, 6.228996753692627, 5.885436058044434, -4.264412879943848, -12.50953197479248, -8.075355529785156, 9.2997407913208, 12.343684196472168, 16.322795867919922, 20.612333297729492, 3.1596438884735107, -7.719261646270752, -7.962483882904053, 15.74555492401123, 1.2258256673812866, -0.4950478672981262, -4.995363712310791, 16.651906967163086, -4.952300548553467, 16.359603881835938, 28.319400787353516, -21.956501007080078, 3.44425630569458, 17.958406448364258, 15.696606636047363, 1.3991494178771973, 9.912894248962402, 0.16958440840244293, 4.552014350891113, -13.831049919128418, 8.00864028930664, -1.0634822845458984, -12.70588493347168, 3.6296579837799072, -5.021938800811768, 6.635217666625977, -17.484966278076172, 17.033201217651367, 6.355706691741943, -1.3441206216812134, 23.652244567871094, 6.977845668792725, -5.414547443389893, 0.7736455202102661, -16.867874145507812, -6.516383171081543, 8.606695175170898, 8.552282333374023, -13.576117515563965, 10.070141792297363, -9.90262222290039, -24.26069450378418, -25.214027404785156], [11.812594413757324, 33.95217514038086, 3.9052252769470215, -27.954195022583008, 33.56709289550781, 18.058090209960938, 7.509475231170654, -6.937572479248047, -0.8720635175704956, -10.996686935424805, 11.342996597290039, 23.1702823638916, 3.5711803436279297, 21.07969093322754, 12.264429092407227, 15.426711082458496, -9.682083129882812, -2.1140313148498535, 5.284916877746582, 7.461991786956787, -7.911678314208984, 0.6713998913764954, 8.086186408996582, 4.8275017738342285, -32.180301666259766, -7.2254486083984375, 15.244192123413086, -15.985038757324219, 28.80829429626465, 10.232149124145508, -17.713199615478516, -7.526383399963379, 21.095184326171875, 22.002012252807617, 25.06170082092285, 3.0757248401641846, -8.548794746398926, -7.189333915710449, -3.8141984939575195, 18.195425033569336, -31.573923110961914, 6.473751544952393, -42.13137435913086, -2.580207109451294, -14.369560241699219, -2.9322409629821777, -43.903236389160156, 10.908747673034668, 27.241321563720703, 2.955172538757324, 33.566349029541016, -17.164875030517578, -36.74310302734375, 32.812530517578125, -0.8482729196548462, -12.977407455444336, 16.01520347595215, 16.125715255737305, 4.205748558044434, 3.0580313205718994, -14.718827247619629, -8.917159080505371, -11.355326652526855, -9.16854476928711], [-26.54254150390625, -4.929806709289551, 6.426823139190674, 11.5186128616333, -3.9559311866760254, 14.046835899353027, -7.407832145690918, 24.930801391601562, 11.722764015197754, 11.671862602233887, -17.771154403686523, -41.41933822631836, 3.0519180297851562, 5.4745941162109375, -7.723170757293701, -30.06983757019043, -6.877959728240967, 4.602622032165527, 5.125164985656738, 2.688894271850586, 2.1736788749694824, 16.896381378173828, -35.45011520385742, -9.21789264678955, 11.561104774475098, 25.295682907104492, -28.64482879638672, 20.920547485351562, -4.191345691680908, 3.4361462593078613, -3.845935583114624, -13.214522361755371, -19.647851943969727, 10.705334663391113, -12.786231994628906, 18.020544052124023, 50.4842643737793, 2.968360662460327, 13.69306755065918, 6.972487449645996, -29.935556411743164, 8.880685806274414, 24.338165283203125, -10.611467361450195, 21.6988525390625, -2.4267611503601074, -6.936079025268555, 3.0025737285614014, -4.576989650726318, 1.9375590085983276, -14.152558326721191, 1.2203903198242188, 27.74349021911621, 5.988254070281982, 13.18285083770752, -19.0926513671875, -11.220419883728027, 1.6882919073104858, -48.03431701660156, 5.442911624908447, -1.2054367065429688, 6.176511287689209, -18.93324089050293, -4.703293323516846], [-10.109296798706055, -15.948116302490234, 15.085617065429688, 11.135573387145996, -17.142637252807617, 11.208820343017578, 14.06657600402832, -0.2503978908061981, -5.725719928741455, -38.029441833496094, 11.837270736694336, 1.9572912454605103, 9.565171241760254, 11.769247055053711, 0.4763079583644867, -7.384554862976074, -9.220809936523438, -4.31147575378418, 14.51626205444336, -4.296582221984863, -3.1699962615966797, -25.975391387939453, 18.675764083862305, 26.278024673461914, -2.3299922943115234, 9.50075626373291, 1.665666937828064, 9.057056427001953, -17.278894424438477, -9.467257499694824, 21.855518341064453, -18.357017517089844, 11.383392333984375, -17.083301544189453, 19.8387508392334, 1.897743821144104, 16.64717674255371, -0.5845360159873962, 5.1226325035095215, 31.704357147216797, 1.309586524963379, 15.415979385375977, 33.45099639892578, -7.575133800506592, -17.290359497070312, -2.086168050765991, 25.480531692504883, 25.414398193359375, -1.341119647026062, 15.92525577545166, -7.692173004150391, -6.036153793334961, 12.511659622192383, 20.192291259765625, 12.056488037109375, -0.07765166461467743, -20.678977966308594, -36.67295837402344, -9.247295379638672, -4.400024890899658, 5.3638081550598145, -5.8952789306640625, 18.231332778930664, -5.150130748748779]], [[6.9641289710998535, -17.49799346923828, -11.075570106506348, 12.987785339355469, -1.0028905868530273, 20.676109313964844, 32.57039260864258, -17.817358016967773, -0.870538592338562, 1.803317904472351, 12.21249008178711, -25.9937801361084, -5.8957319259643555, 5.217369079589844, 9.00535774230957, 2.5871753692626953, 44.77214050292969, -7.699128150939941, 19.65439796447754, -13.80362606048584, 23.185550689697266, -12.720966339111328, -8.491195678710938, -1.1429706811904907, -35.52980041503906, 11.606793403625488, 14.184429168701172, -36.084938049316406, -0.9058103561401367, 24.067617416381836, -10.587555885314941, 18.465051651000977, -21.03666877746582, -23.935211181640625, -7.050481796264648, -1.4030790328979492, 4.617553234100342, -9.225146293640137, -14.310073852539062, 6.1421217918396, -4.175304889678955, 14.894450187683105, -2.350896120071411, -20.527788162231445, -0.9015442132949829, -19.34892463684082, 3.4226160049438477, 37.537818908691406, -33.83906555175781, 6.491565704345703, -10.021103858947754, -3.098407506942749, -10.038721084594727, -4.011241436004639, 7.671655654907227, 2.3821706771850586, 8.851689338684082, -18.123607635498047, 10.206817626953125, 9.098206520080566, -18.7415828704834, 12.548850059509277, 11.8603515625, -0.2835506796836853], [1.662365436553955, -17.394010543823242, -18.181486129760742, -13.049924850463867, -2.7676963806152344, 10.835905075073242, -16.262229919433594, 14.685124397277832, -43.06620407104492, 4.169286727905273, 5.407347679138184, 6.463208198547363, 5.5173563957214355, 27.21843910217285, -19.578615188598633, -38.144142150878906, -14.46632194519043, 11.068185806274414, 23.655189514160156, -12.098784446716309, 5.48685359954834, 13.651605606079102, 26.265533447265625, -6.912016868591309, 10.398529052734375, -5.349271774291992, 0.06049814075231552, 12.8331937789917, -14.098093032836914, 7.818188667297363, 3.2684714794158936, 34.61197280883789, 28.247987747192383, -8.471176147460938, -13.590917587280273, -9.977187156677246, 4.102287769317627, 19.450510025024414, 7.899548053741455, -26.482898712158203, -27.725570678710938, -11.89389419555664, -4.739635944366455, -8.477279663085938, 29.74448585510254, -5.278730392456055, -8.24385929107666, 9.164007186889648, -6.232945442199707, -25.460262298583984, 1.0757005214691162, -24.863914489746094, -19.561614990234375, 10.263338088989258, 1.3082821369171143, 3.3294947147369385, -34.28585433959961, 26.983728408813477, 9.945286750793457, -11.921792984008789, -16.673751831054688, 9.409082412719727, 6.761052131652832, -24.671281814575195], [-23.123559951782227, 12.440232276916504, 7.136659145355225, -24.77582359313965, -6.3673787117004395, -18.294021606445312, 0.07933513820171356, 8.654912948608398, 4.6459550857543945, 10.478930473327637, 10.950765609741211, -8.418654441833496, 12.152810096740723, -3.497525691986084, 30.703716278076172, 42.27397918701172, -19.686988830566406, 17.181669235229492, 14.356365203857422, -14.049128532409668, 8.192012786865234, 1.6644035577774048, -11.125103950500488, 2.648559093475342, 24.224523544311523, 1.3913750648498535, 23.786457061767578, 22.222997665405273, 14.904153823852539, -3.889531373977661, -8.181929588317871, -34.54059600830078, 11.585736274719238, 12.211457252502441, -8.8584566116333, 11.332962989807129, -5.3872857093811035, -9.521540641784668, 24.18759536743164, 0.5117064714431763, 23.370885848999023, 15.635590553283691, 22.058801651000977, -3.2986199855804443, -16.364912033081055, -4.229557514190674, -16.518577575683594, 12.281538963317871, 13.037296295166016, 12.538799285888672, -7.800137042999268, -16.950069427490234, -31.667625427246094, -19.297117233276367, 23.833839416503906, 26.83559799194336, 18.66175079345703, 3.8812100887298584, 23.219541549682617, -1.0103950500488281, 15.177107810974121, -19.964311599731445, 1.6803170442581177, -21.587587356567383], [18.868988037109375, 6.118436813354492, -30.123233795166016, -28.500898361206055, -14.165853500366211, 7.204260349273682, 1.7311675548553467, 12.190187454223633, 25.463125228881836, 5.592081069946289, 15.596151351928711, -1.008101224899292, -3.3581647872924805, 1.251245379447937, -8.461024284362793, -5.139286041259766, -18.451541900634766, 43.63871765136719, -14.972238540649414, 6.733426570892334, -1.2631657123565674, -11.682228088378906, -0.1514217108488083, 23.880807876586914, 12.990273475646973, 2.1094462871551514, 34.220298767089844, 4.189698696136475, -19.227798461914062, 4.472221851348877, 21.45144271850586, -13.933594703674316, -16.19696617126465, -31.716083526611328, 29.933103561401367, 12.762232780456543, 18.55925178527832, -14.851776123046875, -0.7024913430213928, 8.808347702026367, 13.893513679504395, 2.0016961097717285, 30.674205780029297, -6.6708149909973145, 34.71609878540039, -9.730096817016602, -7.606080055236816, -1.0221645832061768, 47.52066421508789, -10.436132431030273, 4.16090726852417, -25.050613403320312, 14.651639938354492, 3.668800115585327, 19.467227935791016, -35.99566650390625, 15.51484489440918, -16.451810836791992, -3.205477476119995, -1.655678629875183, -12.633138656616211, 2.7830970287323, 12.810242652893066, 10.535072326660156], [12.080533981323242, 11.556245803833008, -3.0589635372161865, 7.407876014709473, 2.857243537902832, -11.835233688354492, 37.31422424316406, 5.828675746917725, -14.16150188446045, 19.981294631958008, 5.580875396728516, 21.99785041809082, 3.8640854358673096, -2.736875057220459, -23.62203025817871, 31.855480194091797, 27.282670974731445, -17.650651931762695, 0.13254767656326294, 26.54135513305664, 16.09503936767578, -14.016478538513184, 26.63279914855957, -9.883502006530762, -5.206966400146484, -16.62473487854004, 23.941041946411133, -9.75334358215332, 13.719091415405273, -25.660120010375977, -21.618066787719727, -13.413049697875977, -0.9523704648017883, 34.343196868896484, 11.33625316619873, -7.410495281219482, -15.502477645874023, -18.862058639526367, -24.533937454223633, 26.41126251220703, -1.489770770072937, 5.488705158233643, -7.904329776763916, 23.24977684020996, -10.40992546081543, -15.736136436462402, -8.146719932556152, 5.938592910766602, 17.94264793395996, 4.869798183441162, -13.861039161682129, -58.22607421875, -7.331356048583984, -13.845109939575195, -16.153921127319336, 8.068554878234863, 33.6192512512207, 4.585381507873535, 23.463706970214844, 14.80214786529541, 13.091289520263672, -9.104642868041992, -10.729072570800781, -5.7161970138549805], [-6.591964244842529, -12.134325981140137, 21.434368133544922, -29.854482650756836, 14.178642272949219, -26.030439376831055, -1.7320857048034668, 15.731672286987305, 12.102073669433594, 4.468930721282959, -30.030269622802734, -7.348806858062744, 11.355799674987793, -4.6704559326171875, -21.756189346313477, -32.5763053894043, 2.6636757850646973, 1.9225356578826904, 24.802881240844727, 2.2194316387176514, 16.855606079101562, 16.27495002746582, -15.077876091003418, -8.638829231262207, 6.810606956481934, 18.25222396850586, 10.323123931884766, -14.010679244995117, -16.606216430664062, 13.267966270446777, -2.9501893520355225, 18.53829002380371, 10.356292724609375, 25.95315170288086, -7.767457008361816, -24.253328323364258, -12.858075141906738, 13.80052661895752, -14.885835647583008, 24.020503997802734, -9.666372299194336, -1.3622454404830933, 17.725400924682617, 3.904568910598755, 14.031085014343262, 26.659027099609375, -6.632718086242676, -13.811022758483887, -2.2525806427001953, 21.279064178466797, -4.593682289123535, 0.671110987663269, -4.979706764221191, 2.2103049755096436, 4.17362117767334, 30.132200241088867, -30.01136589050293, -5.443998336791992, -22.602005004882812, -1.3616758584976196, 19.027833938598633, 6.318418502807617, -14.542472839355469, -19.626087188720703], [7.757160663604736, -4.42643928527832, -6.519266128540039, 11.244466781616211, -12.541271209716797, -4.384144306182861, 7.160589218139648, -14.49930477142334, 29.614919662475586, 1.8539347648620605, 5.3546223640441895, 18.608642578125, 4.783318996429443, -7.13807487487793, -10.506608963012695, -24.069046020507812, -31.412450790405273, -6.495512962341309, -10.490514755249023, 21.071434020996094, 53.286895751953125, -10.132559776306152, -24.815454483032227, 6.615416526794434, -0.4514988660812378, -5.8982672691345215, 11.482933044433594, -18.79645538330078, -10.171039581298828, 14.938204765319824, -15.480374336242676, 7.140304088592529, -17.37238883972168, -18.162479400634766, 12.660436630249023, 39.6943244934082, -25.806489944458008, 0.2650180160999298, -1.2097764015197754, 44.32313919067383, 1.515303373336792, 13.850679397583008, 55.67464065551758, -7.090548038482666, 25.580568313598633, -11.35427188873291, -6.528569221496582, 3.183053970336914, 31.554750442504883, 2.4853010177612305, -14.156214714050293, 13.34601879119873, -14.82084846496582, -18.594161987304688, -20.217748641967773, 2.339801788330078, -3.9467732906341553, -7.265011310577393, -9.816112518310547, -8.06088924407959, -24.09963607788086, 15.915145874023438, -19.905412673950195, 4.060522556304932], [19.59044647216797, -12.694103240966797, 14.957365989685059, -3.349735975265503, 12.326055526733398, -10.660651206970215, 6.354590892791748, 42.844879150390625, -14.396910667419434, -6.755672454833984, -0.2772469222545624, 32.508663177490234, 35.80301284790039, 4.255578517913818, -19.489370346069336, -42.96226119995117, 17.2226505279541, -28.080724716186523, -18.25025749206543, 30.37330436706543, 31.6173038482666, 15.593100547790527, -19.114070892333984, 7.025715351104736, 13.627782821655273, 4.976404190063477, -0.09147115051746368, 40.70777130126953, -33.953678131103516, 17.971546173095703, 9.362737655639648, 18.160228729248047, -5.4410319328308105, 6.11987829208374, -11.26109790802002, -14.132108688354492, 33.33125686645508, 20.45138931274414, 3.4139113426208496, -14.414374351501465, -10.525472640991211, -16.294109344482422, -9.180947303771973, 9.11062240600586, 20.703744888305664, -6.0371270179748535, 3.8568670749664307, 25.22113609313965, -11.23776912689209, -30.033967971801758, 8.292672157287598, -9.556029319763184, 15.578977584838867, 27.648283004760742, 7.95601224899292, -17.321889877319336, -18.047515869140625, -13.562610626220703, -12.126266479492188, -14.916756629943848, 9.643352508544922, -21.870807647705078, 41.017391204833984, -14.647562026977539], [-6.340792179107666, -0.629062294960022, 6.897221088409424, -7.063351154327393, -22.18606948852539, 3.8859286308288574, -8.610179901123047, -1.3787033557891846, 3.349393367767334, -1.4939684867858887, -17.850719451904297, -2.524686098098755, 12.905815124511719, 4.760385513305664, -4.648507595062256, -7.33178186416626, -5.242616653442383, 16.4049072265625, 20.53713035583496, -6.780535697937012, -8.251914978027344, 21.2093448638916, 4.706111431121826, -0.9920336604118347, 19.486705780029297, -22.191017150878906, 11.840032577514648, 22.39504623413086, 0.4254491329193115, 0.9262411594390869, -7.58404541015625, 2.9481136798858643, -4.404433727264404, 22.473651885986328, -12.415934562683105, -14.003594398498535, -14.776922225952148, 30.097707748413086, 27.224178314208984, -2.2255022525787354, -2.397897481918335, 20.224458694458008, 7.865498065948486, -37.82441711425781, 20.0166072845459, 17.131101608276367, 0.0469241701066494, -5.982375621795654, -6.318447589874268, 2.9391839504241943, 21.06171417236328, 1.052955985069275, 2.55051326751709, 10.502622604370117, 16.817628860473633, -24.249006271362305, -2.8756861686706543, 12.151930809020996, -2.7898690700531006, -2.2532851696014404, 20.04500961303711, -17.869701385498047, 5.356252670288086, -2.4832894802093506], [-29.39607810974121, -38.1146354675293, 11.454130172729492, 19.26816749572754, -4.624634742736816, 10.695398330688477, 16.786890029907227, -9.441691398620605, -11.455413818359375, -14.832535743713379, -1.3937267065048218, 6.266429424285889, 4.498190879821777, 0.6309641003608704, 12.285927772521973, 18.3348445892334, 25.520803451538086, -40.11506271362305, 27.7999267578125, 20.845123291015625, 4.900515556335449, -9.383981704711914, 18.556095123291016, -6.9411468505859375, -33.21366882324219, -18.30964469909668, 15.744641304016113, 1.2424407005310059, 9.969490051269531, 10.400132179260254, -27.299848556518555, 2.021117925643921, -0.6854634284973145, -15.908930778503418, -25.485023498535156, 11.052318572998047, -21.266803741455078, 19.488006591796875, 30.3006534576416, -24.720365524291992, -3.5942647457122803, 1.9302932024002075, 9.931788444519043, -4.68784761428833, -0.34418222308158875, 5.964849948883057, 39.975406646728516, -22.07256317138672, 6.719964027404785, -10.706608772277832, 13.281076431274414, 31.613676071166992, 7.181730270385742, -21.54892921447754, -11.239734649658203, -1.4616533517837524, 47.405094146728516, 3.200540065765381, 36.39817428588867, -17.135074615478516, 1.066409945487976, -19.882179260253906, 25.494468688964844, -4.008522033691406], [13.28006362915039, 30.94013786315918, -19.96980094909668, 8.41431713104248, -3.6545193195343018, -0.06765706837177277, 26.5314884185791, -7.0474019050598145, -0.2256447672843933, 16.729175567626953, 8.812447547912598, -11.302921295166016, -37.565589904785156, 9.91220474243164, 24.451101303100586, 9.700281143188477, -2.289830446243286, 13.690152168273926, 12.439407348632812, 5.459962368011475, -11.954679489135742, 17.130640029907227, -22.47450828552246, -9.283248901367188, -6.834541320800781, -11.918045043945312, 20.9021053314209, -5.622227191925049, -14.168020248413086, 11.273898124694824, -12.020909309387207, -3.0453155040740967, 18.489055633544922, -14.540528297424316, 6.241990089416504, -17.42400360107422, 8.01725959777832, -13.4266996383667, 2.040215253829956, -7.95079231262207, 6.061382293701172, 0.5799501538276672, 2.9508843421936035, -11.095043182373047, 0.7416911721229553, 5.972045421600342, 20.051294326782227, 11.458052635192871, -2.757910966873169, -5.153203010559082, -38.3109245300293, -10.48617935180664, 11.867698669433594, 11.469125747680664, -1.6606680154800415, 17.70575523376465, -7.81911039352417, -23.003921508789062, -11.948019981384277, -19.95824432373047, -9.467497825622559, 5.841719627380371, 6.325510025024414, 7.664602279663086], [13.929030418395996, 18.004961013793945, -13.874963760375977, -0.09749004244804382, -16.122472763061523, -10.231374740600586, -6.671353340148926, 21.40229606628418, -41.26844787597656, 4.917988300323486, -12.957275390625, 10.344610214233398, -15.032898902893066, 0.5018240213394165, -1.8062446117401123, 9.622788429260254, -28.95047378540039, 4.459073543548584, -28.10657501220703, 5.561806678771973, -25.737722396850586, 15.238852500915527, -17.79314613342285, -11.710617065429688, 2.887566328048706, -19.570180892944336, -5.004273414611816, 27.20884132385254, -6.506117343902588, 6.563220024108887, -3.8813977241516113, -16.494470596313477, -15.845474243164062, -3.249720573425293, 24.883075714111328, 33.291481018066406, 25.3769588470459, 2.6140201091766357, 40.29485321044922, -23.00029945373535, 15.14949893951416, 13.59705638885498, 2.207402467727661, -1.7578908205032349, -24.028461456298828, -12.362103462219238, 22.089719772338867, 36.17616271972656, -0.6555637717247009, -24.49321174621582, 7.1637349128723145, 1.196597933769226, 1.8557376861572266, -5.741872787475586, 5.476706504821777, -39.082698822021484, -7.422381401062012, 23.805025100708008, -9.823690414428711, 11.80846881866455, 3.5461785793304443, 3.3894460201263428, 1.377723217010498, -14.300580024719238], [-5.802216529846191, 13.845536231994629, -19.624685287475586, 14.587794303894043, -9.391938209533691, -3.9931721687316895, 17.663076400756836, -13.62053108215332, 22.06500244140625, -18.75311279296875, 24.259780883789062, 10.851597785949707, -1.7083879709243774, -7.78583288192749, 8.171124458312988, 4.759492874145508, -10.912202835083008, 11.105718612670898, 9.579097747802734, 14.076482772827148, -3.00003719329834, -11.53612232208252, -6.227724075317383, -27.161039352416992, 36.660552978515625, 29.35594367980957, 16.973926544189453, 7.721739292144775, 31.309246063232422, 0.9198412895202637, 12.432308197021484, 23.470172882080078, -4.543311595916748, 18.83209991455078, 1.1330499649047852, -17.61623191833496, -3.914616107940674, 7.694786548614502, 11.996295928955078, 21.170549392700195, -1.357387900352478, -30.59221076965332, 11.780171394348145, -17.560516357421875, -29.260135650634766, -0.39946249127388, 2.8234682083129883, 7.321136474609375, -3.5943892002105713, 9.581918716430664, 5.733223915100098, 0.5697195529937744, 0.6541686058044434, 9.329968452453613, 16.076635360717773, -18.959951400756836, -27.156930923461914, 25.953922271728516, -39.97328567504883, -18.036638259887695, 1.9149583578109741, -22.183183670043945, -39.28525924682617, 11.97792911529541], [15.481099128723145, -10.468690872192383, -49.787757873535156, 11.782440185546875, 17.594846725463867, -4.467092514038086, -0.6618288159370422, -17.94500160217285, 7.333745956420898, -1.8580266237258911, -28.850326538085938, -6.438957214355469, 26.218366622924805, -29.296192169189453, 30.886201858520508, 19.45621681213379, 35.19441604614258, -17.6700496673584, 14.998220443725586, 34.819061279296875, 29.777902603149414, -8.063385009765625, -25.959043502807617, -19.855133056640625, 13.113336563110352, -19.203819274902344, 18.161319732666016, -13.984813690185547, -3.5255494117736816, 8.240418434143066, -27.49997329711914, -3.1893420219421387, -22.6363468170166, 1.8685842752456665, -0.6350300312042236, -4.357918739318848, -8.010560989379883, 4.1700520515441895, -0.0061599756591022015, -6.818770408630371, 21.839529037475586, 22.684572219848633, 7.109403610229492, 7.852732181549072, 14.530357360839844, 18.000938415527344, 16.338350296020508, -17.649192810058594, 13.190234184265137, -1.4703714847564697, 4.541163921356201, 12.804004669189453, 11.14552116394043, -12.873353958129883, 8.419868469238281, -2.0869975090026855, -20.85367774963379, -24.854278564453125, 9.810505867004395, 37.0513916015625, 19.93466567993164, 2.7228405475616455, 15.571914672851562, 0.40864548087120056], [1.062080979347229, 18.925626754760742, -2.967174768447876, 0.3105291724205017, -2.5612449645996094, -0.054382871836423874, 9.783222198486328, 20.487796783447266, -2.3071043491363525, 0.04491710290312767, 5.716953277587891, -2.2999589443206787, -18.88700294494629, 1.7542778253555298, -32.49332809448242, -18.426061630249023, 31.733081817626953, -8.896574020385742, -9.659281730651855, 9.562801361083984, -15.272010803222656, -12.080228805541992, 19.672344207763672, -8.128875732421875, 1.9313418865203857, -28.13835334777832, -5.383853435516357, -14.014219284057617, -18.112995147705078, 2.22178053855896, 6.7214202880859375, 5.062222957611084, -14.763602256774902, -1.7119513750076294, 13.111207008361816, 15.163768768310547, 5.463789463043213, 24.42731285095215, 23.246042251586914, -13.826224327087402, 4.863167762756348, 6.795923709869385, 8.48055648803711, 4.573704242706299, 33.04245376586914, 5.652194976806641, -24.80552101135254, 8.440650939941406, 1.114709496498108, -4.698007583618164, 24.21408462524414, -18.788541793823242, 6.602456092834473, -6.128626823425293, -0.03923264518380165, -14.235254287719727, -40.04954147338867, -7.250478744506836, -7.168064117431641, 10.269474983215332, 0.9346094131469727, -10.342510223388672, 20.85376739501953, 3.5687198638916016], [-32.49523162841797, -11.423850059509277, -1.932280421257019, 4.188014507293701, -5.035221099853516, 30.022401809692383, -6.1364521980285645, -15.545682907104492, 2.17828369140625, -9.012826919555664, -22.220949172973633, -30.944536209106445, 9.3975830078125, 9.800103187561035, -28.3305721282959, 22.04587745666504, -0.8390706777572632, 23.19134521484375, -30.594308853149414, -1.557550311088562, -6.611203193664551, 21.325132369995117, 4.853114604949951, 11.556360244750977, -8.040027618408203, 5.76365852355957, 16.448131561279297, 13.37656307220459, 6.285294532775879, 15.699883460998535, -1.8858832120895386, -25.303836822509766, 30.204444885253906, -12.4952974319458, -24.026090621948242, -11.337250709533691, -0.11899904906749725, 18.28133773803711, -20.50783348083496, -0.9821785092353821, -13.452957153320312, -14.656627655029297, -6.294559001922607, 1.6286035776138306, -15.457341194152832, -11.031929969787598, -2.44024395942688, -11.511631965637207, -9.79081916809082, -5.841982841491699, 17.89520835876465, 1.7712424993515015, 7.548572063446045, -1.5584144592285156, -19.924726486206055, 21.699811935424805, 16.217336654663086, 2.9643197059631348, 10.273816108703613, 9.258503913879395, -4.989543437957764, 2.6835172176361084, 11.322480201721191, 8.485404014587402]], [[-2.975792169570923, -9.818017959594727, -1.2091712951660156, 4.596524238586426, -32.589561462402344, 7.111673355102539, 2.734891653060913, -5.892330646514893, -15.115979194641113, -22.750593185424805, 15.085391998291016, -21.666656494140625, -9.510444641113281, 2.760403871536255, -2.4898781776428223, 10.676787376403809, -10.139413833618164, -3.2116222381591797, 12.527814865112305, -24.295909881591797, 1.0869624614715576, -35.401710510253906, 4.810050964355469, 22.97860336303711, -9.854604721069336, 31.782623291015625, 9.683097839355469, -3.776156425476074, 2.7587122917175293, 1.3538602590560913, 10.571534156799316, -3.1737864017486572, 46.7203254699707, -3.4513957500457764, -2.6378002166748047, -5.651208877563477, -6.328039646148682, -2.2621617317199707, 38.39295959472656, 0.18857555091381073, 0.1350562870502472, -1.9786746501922607, -1.525535225868225, -7.668128490447998, -0.3933524489402771, -2.913835048675537, -22.02298927307129, -29.377662658691406, -19.218969345092773, -5.608408451080322, 17.42169189453125, 18.3171443939209, 5.1531243324279785, -18.57561492919922, -7.507443428039551, -19.972333908081055, 44.63520812988281, -36.36150360107422, 5.353227615356445, 9.001757621765137, -36.420528411865234, -7.473638534545898, 15.090028762817383, 24.293519973754883], [4.768006324768066, -1.5030593872070312, 18.264020919799805, -24.642484664916992, 24.692665100097656, 15.947107315063477, 30.23911476135254, -11.47410774230957, -7.375419616699219, 7.811120986938477, -10.769850730895996, 5.0553812980651855, -14.960858345031738, -1.5971170663833618, 29.311725616455078, 10.732815742492676, 24.818349838256836, -20.48459815979004, -9.44094181060791, 26.020526885986328, 19.306026458740234, -3.124727725982666, 2.611610174179077, 16.781558990478516, 7.462025165557861, 7.361189842224121, -10.561192512512207, -26.93463706970215, 6.255102157592773, 2.891002893447876, -9.002985000610352, -4.410602569580078, 8.756726264953613, 14.78226089477539, -21.350820541381836, -25.96220588684082, -10.20665454864502, 3.259425163269043, 10.12472915649414, 0.6961209177970886, -12.496850967407227, 22.660736083984375, 12.419273376464844, -2.367781639099121, -6.508650302886963, -29.602699279785156, 3.9945645332336426, 7.4825053215026855, -0.10211959481239319, -20.372753143310547, -28.610231399536133, 8.629612922668457, 3.4476852416992188, 21.88264274597168, -10.185924530029297, 1.7687875032424927, -16.301172256469727, -2.4663496017456055, -20.398664474487305, 14.58869743347168, 16.13684844970703, -16.42125701904297, -2.592820167541504, 5.626154899597168], [-3.255572557449341, 15.422586441040039, 14.619935035705566, 6.929007053375244, -14.297706604003906, -8.901627540588379, -15.622578620910645, 10.429535865783691, 4.944191932678223, -0.9903789162635803, -19.474090576171875, 1.1402734518051147, -3.1079261302948, -2.1411497592926025, -13.784807205200195, 20.794700622558594, -21.591167449951172, 23.772459030151367, 6.3755693435668945, -4.515806674957275, -19.4786434173584, -24.725515365600586, 8.271854400634766, 34.41436767578125, 7.336341857910156, -5.310824394226074, 1.3988077640533447, -7.949074745178223, 18.534324645996094, -1.2271054983139038, 18.49982452392578, 17.409090042114258, -14.197392463684082, 14.920653343200684, 8.85308837890625, -16.65362548828125, -19.943580627441406, 31.360361099243164, 0.9188598990440369, -8.483942031860352, -10.656371116638184, 36.38909149169922, -2.708904266357422, -0.2705609202384949, 8.808065414428711, -7.423357009887695, 4.4324564933776855, 0.7811635136604309, -2.555781126022339, 6.36102819442749, -14.523778915405273, -7.526810169219971, -15.86363410949707, 18.408512115478516, 0.9813442230224609, -1.706739902496338, -14.765090942382812, 3.136033296585083, -16.506650924682617, 4.484081268310547, 12.256587982177734, 15.410036087036133, -14.876800537109375, 14.748697280883789], [13.015959739685059, -30.214262008666992, -2.43023943901062, -26.692935943603516, -1.4672244787216187, 10.162245750427246, 10.88814926147461, -19.81210708618164, 36.883018493652344, -15.695971488952637, 22.841978073120117, -11.931849479675293, 29.608909606933594, 22.811796188354492, 12.070195198059082, -4.90998649597168, 1.5143011808395386, 6.7704243659973145, 0.7402714490890503, -15.483716011047363, -12.952006340026855, 1.6580406427383423, 0.8983638882637024, -3.8904528617858887, 12.145661354064941, -4.216198444366455, 40.98989486694336, 7.709512233734131, 9.270086288452148, -24.900157928466797, 1.399057149887085, 10.902914047241211, -19.905107498168945, 10.33474349975586, -21.55162811279297, -6.80930233001709, -4.410546779632568, 11.253135681152344, -4.892125606536865, -21.362253189086914, 10.376618385314941, 16.263381958007812, 7.200798988342285, -4.944336414337158, 0.8946031928062439, 4.421740531921387, 7.412100791931152, 4.618541240692139, 11.810297966003418, -13.373725891113281, 23.63337516784668, 17.593486785888672, -1.3517571687698364, -12.577024459838867, 7.598036289215088, 19.60972785949707, -22.01325798034668, -3.8443408012390137, -3.929509401321411, -11.764394760131836, 1.6794098615646362, -10.861701965332031, -19.803552627563477, 16.526973724365234], [-20.768800735473633, -17.391006469726562, -6.632778644561768, 7.201748847961426, -9.887125968933105, 21.56140899658203, -36.18047332763672, -15.8794584274292, 21.390892028808594, -44.64570999145508, 18.391010284423828, -12.13027572631836, 8.572667121887207, -13.212220191955566, 7.742873668670654, -13.178810119628906, 11.89625358581543, -16.569772720336914, 0.3089964985847473, -16.90827751159668, 11.259706497192383, -10.193967819213867, 19.678020477294922, -7.3855977058410645, 10.264496803283691, -19.871009826660156, 10.080928802490234, 2.450498580932617, 11.068288803100586, -0.8174563050270081, 8.69877815246582, -4.99166202545166, 8.430312156677246, 22.559173583984375, -29.760303497314453, -5.47226095199585, 10.573623657226562, -30.593740463256836, -1.3972991704940796, 5.441981792449951, -8.610709190368652, -18.537765502929688, 4.1775689125061035, 22.87491798400879, 10.645387649536133, 10.034567832946777, 10.483230590820312, 19.820011138916016, -17.23772430419922, -22.475709915161133, -31.7939510345459, 15.589069366455078, -18.759607315063477, -17.122867584228516, 10.352869987487793, 11.937840461730957, -7.908321380615234, -6.7778449058532715, -24.889057159423828, 12.915205001831055, 11.746938705444336, 15.851960182189941, -12.616167068481445, 4.146190166473389], [14.16588020324707, -10.026619911193848, 13.492063522338867, -6.539459228515625, -6.328539848327637, -17.645790100097656, 6.712924003601074, -15.20452880859375, 28.223501205444336, 19.204437255859375, -0.4968098998069763, 1.3094801902770996, -13.904348373413086, 0.8036473393440247, -5.696917533874512, 8.38992691040039, 11.681593894958496, -18.478960037231445, -0.2948842942714691, -19.17044448852539, -5.256664752960205, -6.1164140701293945, -29.476375579833984, 2.642643928527832, -2.2339696884155273, 12.183316230773926, 7.680534362792969, 23.800065994262695, 5.735558986663818, 17.430940628051758, -20.01900291442871, 12.783921241760254, -3.913722038269043, 23.887229919433594, -17.702953338623047, -3.0740809440612793, -42.41763687133789, -12.134183883666992, -17.595584869384766, 11.313821792602539, -14.371582984924316, -4.157401084899902, 6.066410541534424, 2.584296226501465, 6.883066654205322, 0.8316466212272644, -29.990991592407227, 3.20804762840271, -3.0010807514190674, -4.005300521850586, -8.207930564880371, -6.257638931274414, 24.413068771362305, -6.557075023651123, -22.714252471923828, 2.294904947280884, 20.190183639526367, -3.578843832015991, 7.922215461730957, 14.71205997467041, -13.01394271850586, -22.238039016723633, -12.890332221984863, -6.052952766418457], [13.6658353805542, -25.739437103271484, -4.1787190437316895, -8.98851490020752, 8.620092391967773, -8.9972562789917, 7.579325199127197, -8.855956077575684, 14.510611534118652, 5.2077317237854, -6.669987678527832, -2.474579095840454, 19.28908348083496, -7.91317081451416, 10.477516174316406, 12.136932373046875, 20.96355438232422, -22.838665008544922, -20.30988311767578, -25.599912643432617, -5.40116548538208, 5.467200756072998, 27.451194763183594, 15.707913398742676, 10.149869918823242, -4.885939121246338, -11.197307586669922, 6.214330673217773, 22.768722534179688, -10.636521339416504, 20.128808975219727, -1.8991687297821045, -10.265501976013184, 3.291348695755005, -0.13671477138996124, -18.236995697021484, 6.5655436515808105, 10.782899856567383, -24.502696990966797, -21.295949935913086, -23.9057674407959, 35.19153594970703, -18.964345932006836, -42.47990417480469, -13.942021369934082, 17.130033493041992, 28.609270095825195, 31.109567642211914, 9.370068550109863, -16.11989974975586, 11.649958610534668, 35.360774993896484, -20.54467010498047, -1.5661017894744873, -10.444635391235352, 21.82884979248047, -0.27515527606010437, 3.783003330230713, -6.743119239807129, 32.86524963378906, 1.0885964632034302, -16.242143630981445, 0.8423159122467041, 9.798783302307129], [-27.992229461669922, -21.343982696533203, -8.175219535827637, 22.438291549682617, -29.81605339050293, 9.432056427001953, 8.611323356628418, -6.362547874450684, -6.077850818634033, -11.490828514099121, 19.800792694091797, 14.226018905639648, -2.2130610942840576, -18.935447692871094, 21.660781860351562, 25.458385467529297, -4.058231830596924, -8.787321090698242, -6.971073627471924, 8.34449577331543, -17.192960739135742, -1.8548880815505981, 20.055049896240234, 33.600460052490234, -1.4244860410690308, 2.4154176712036133, 19.11083984375, 23.871105194091797, 24.321212768554688, -12.039580345153809, -9.109447479248047, 5.245315074920654, 8.736894607543945, -3.4321861267089844, -17.45435333251953, 25.28076934814453, 12.52526569366455, 11.624189376831055, 4.88234281539917, -4.69639778137207, 16.28424072265625, 13.542320251464844, 41.711727142333984, -1.758631944656372, -14.874191284179688, -7.559220314025879, 30.81393814086914, -9.364314079284668, -3.3175928592681885, 4.709574222564697, 2.347161054611206, -4.520136833190918, 5.1799750328063965, -9.99514389038086, -23.9352970123291, -9.599221229553223, 15.034825325012207, -10.172664642333984, 10.043094635009766, 4.139265537261963, -15.305055618286133, 16.250879287719727, 29.46018409729004, 22.18169593811035], [-9.360275268554688, -15.161587715148926, 0.7866319417953491, 1.4705884456634521, 24.696256637573242, -15.498798370361328, -2.5456409454345703, 10.61876392364502, 9.58630657196045, 11.085932731628418, -3.4369421005249023, -2.5601327419281006, -10.991231918334961, -18.4194278717041, -6.61378288269043, 9.418703079223633, -7.516376972198486, -15.425569534301758, 2.5124409198760986, -18.123767852783203, -18.304983139038086, -12.488179206848145, 1.1257867813110352, -11.040040969848633, 5.5607805252075195, -3.296541690826416, 29.291593551635742, -7.7163286209106445, -26.683053970336914, -0.9658917784690857, -4.588732719421387, -8.857312202453613, 1.210924506187439, -20.800844192504883, -16.84466552734375, -6.022989273071289, -7.466548442840576, 6.6670026779174805, 21.258209228515625, 21.872854232788086, 13.447860717773438, -21.495262145996094, -2.0791261196136475, -1.1208652257919312, 9.000035285949707, 28.05953025817871, 10.645360946655273, -12.614012718200684, -1.2666271924972534, 2.922057628631592, 18.617755889892578, 24.356645584106445, 6.664348602294922, -24.920730590820312, -4.263708591461182, -2.9410576820373535, -17.577285766601562, -7.041038513183594, -1.0182933807373047, 0.5737610459327698, -13.652490615844727, 3.480961799621582, 17.452587127685547, 0.8106638193130493], [8.719820022583008, 5.14956521987915, -6.033082008361816, -3.5183651447296143, 10.541247367858887, -1.048838496208191, -18.983327865600586, 20.17724609375, -14.928943634033203, 6.032675743103027, -8.026655197143555, -6.336691856384277, 2.6219964027404785, -0.35349538922309875, 4.818789958953857, -21.68386459350586, -21.947832107543945, -8.0789794921875, 0.46891674399375916, 17.153383255004883, 46.64890670776367, -4.524289608001709, -3.370595693588257, 22.42817497253418, 34.31707763671875, 8.806368827819824, -9.009718894958496, 13.3117094039917, -37.153106689453125, 10.217610359191895, -7.887467861175537, 7.10508394241333, -5.778082370758057, -3.639369249343872, -2.3085734844207764, -10.167884826660156, -3.9015607833862305, -4.338175296783447, -0.5617015361785889, 7.938503265380859, 28.868122100830078, 7.305546760559082, 2.1859161853790283, -5.3882012367248535, 7.888402938842773, 1.2636377811431885, -11.55594539642334, -4.097780704498291, -24.833763122558594, -8.397360801696777, -11.810930252075195, 6.888542175292969, 23.973196029663086, 9.53600788116455, 33.69240951538086, -12.243806838989258, -4.069189071655273, -8.460135459899902, 31.28849220275879, 0.8264384269714355, -8.42038345336914, 5.971080780029297, -27.239816665649414, -30.12688636779785], [6.6015448570251465, -5.814122676849365, 34.54319763183594, 0.9792816638946533, 4.364102363586426, 11.315045356750488, -10.894892692565918, 3.6628258228302, 21.31787872314453, -18.725229263305664, 5.081826686859131, 30.925533294677734, 25.723388671875, 8.405207633972168, 25.960769653320312, -3.424729108810425, -24.672269821166992, 6.8115997314453125, 8.473033905029297, 44.5423583984375, -2.2739596366882324, 8.034196853637695, 2.0796704292297363, 5.739369869232178, -26.130123138427734, 14.424948692321777, -5.6769914627075195, -35.694091796875, 10.597114562988281, -12.740438461303711, -27.61545753479004, 24.371253967285156, -0.3748764395713806, 3.097618579864502, -24.172893524169922, -2.813063621520996, -3.8245911598205566, 17.414037704467773, -14.510183334350586, -10.31795883178711, 22.592857360839844, 17.636260986328125, 11.75404167175293, -4.900228977203369, -1.887770414352417, 3.626443386077881, 4.901598930358887, 4.516075134277344, -1.3702802658081055, 21.469449996948242, 25.036937713623047, 4.686980724334717, -13.115289688110352, 8.35379695892334, -9.897172927856445, 14.122956275939941, -0.6952024698257446, -19.638343811035156, -0.6022344827651978, -16.999221801757812, -3.567392587661743, 19.131553649902344, 15.431619644165039, -9.251214981079102], [-26.717620849609375, -15.892655372619629, 19.270898818969727, -3.102231740951538, -1.585297703742981, -15.468384742736816, 5.237221717834473, 18.044700622558594, 14.975794792175293, -10.596566200256348, -4.179821491241455, 4.55096960067749, 10.516861915588379, -29.215633392333984, 1.4924335479736328, 0.09267222136259079, -12.101639747619629, -8.60306453704834, -11.062886238098145, -3.243394136428833, 36.219635009765625, -21.797080993652344, 7.3840837478637695, 1.3375028371810913, 10.541603088378906, 4.095469951629639, 46.3341178894043, 19.125995635986328, 19.67194175720215, 0.8839177489280701, -23.024314880371094, 10.914029121398926, -1.593300461769104, -2.9262824058532715, 1.6794202327728271, 21.110591888427734, 19.709009170532227, 4.863293647766113, -14.81624698638916, 6.133172512054443, -5.8720269203186035, 18.304744720458984, 11.923537254333496, -4.084418773651123, -12.031390190124512, 23.733028411865234, 41.45209503173828, -4.931948661804199, -8.202444076538086, 17.286785125732422, 20.194202423095703, -6.012582302093506, 10.034966468811035, 3.937922954559326, -0.06778908520936966, -18.454357147216797, -1.4077824354171753, -27.310726165771484, 6.500408172607422, -19.144685745239258, 3.546581983566284, -0.6388627290725708, 4.646509170532227, -18.571657180786133], [-26.155609130859375, -7.30407190322876, 7.650923252105713, -7.433484077453613, 5.594224452972412, 22.43775749206543, -0.4265655279159546, -11.227922439575195, 5.124528408050537, 9.176762580871582, -15.000967025756836, -7.26411771774292, 9.241937637329102, 23.035022735595703, 6.70440673828125, -10.52746295928955, 9.878867149353027, -5.59822416305542, 19.51371192932129, -38.66321563720703, 28.802106857299805, -14.262707710266113, -9.676894187927246, -3.7698235511779785, 30.412811279296875, -7.539843559265137, -6.047476768493652, -24.863582611083984, -20.764575958251953, 5.045224189758301, -20.27094841003418, -12.863100051879883, -38.564903259277344, 32.60345458984375, -32.31208419799805, -18.35751724243164, 35.39767837524414, 28.399900436401367, -6.151861190795898, -9.09367847442627, 11.276885986328125, -16.966411590576172, 9.56344985961914, -11.177741050720215, -25.578060150146484, 5.745344161987305, -5.50000524520874, 11.639538764953613, 18.39605140686035, 1.4823137521743774, 18.018966674804688, -26.06589126586914, 16.330514907836914, -10.961544036865234, -27.85438346862793, -19.207578659057617, -17.872623443603516, 10.89035415649414, -17.71408462524414, 3.4997856616973877, -25.550487518310547, 13.831966400146484, 21.850427627563477, -6.629510879516602], [-6.423604488372803, 31.480066299438477, 14.721182823181152, -17.827011108398438, -7.1726789474487305, -27.897682189941406, -13.971399307250977, -20.94906234741211, 14.373239517211914, -7.606091499328613, -12.660497665405273, -7.283234119415283, -12.42199993133545, -11.122687339782715, 14.679729461669922, 44.80214309692383, -3.4401066303253174, -6.626470565795898, -16.37726402282715, -22.094284057617188, -0.6823858618736267, -14.246596336364746, -43.94424057006836, -4.842390537261963, 4.388419151306152, 17.80554962158203, 0.7564650774002075, 14.333425521850586, -7.986924171447754, 21.984121322631836, 0.5253930687904358, -1.7225539684295654, 12.518708229064941, -21.613765716552734, 4.869841575622559, -26.88276481628418, -13.585514068603516, -7.02519416809082, -9.120809555053711, -6.090160846710205, -26.51831817626953, 16.03900718688965, -5.423105716705322, 2.035994052886963, 11.071046829223633, -28.38352394104004, -14.078518867492676, -3.4975619316101074, 8.707819938659668, -5.979777812957764, -17.172683715820312, -22.382726669311523, -11.580584526062012, -13.330257415771484, 19.031959533691406, -2.381981134414673, -14.496378898620605, 12.04120922088623, 0.9780750274658203, 9.224125862121582, 18.548234939575195, -5.773223876953125, -11.328096389770508, -12.295894622802734], [-21.225404739379883, -27.358182907104492, -5.916242599487305, 14.117962837219238, -17.81020736694336, 7.460728168487549, -9.242816925048828, 6.5404438972473145, 7.04018497467041, 2.5478479862213135, 2.9382357597351074, 3.7492687702178955, 10.925809860229492, -23.016925811767578, -13.52894401550293, -8.228727340698242, -0.3746894896030426, 2.2088475227355957, 8.417488098144531, 17.73749542236328, 5.354800701141357, 16.540935516357422, 0.8326460123062134, -10.184304237365723, -4.327444076538086, 14.299151420593262, 13.86538314819336, 8.804084777832031, 10.669140815734863, -7.925934314727783, -15.087357521057129, 8.425575256347656, -1.3417046070098877, -2.377918004989624, 0.46576616168022156, 26.78461456298828, 20.318056106567383, -14.707496643066406, 11.337119102478027, -17.12858009338379, -2.7560477256774902, 25.47443199157715, -0.04166175797581673, -5.761806011199951, -27.873973846435547, 23.103715896606445, 18.391704559326172, -26.646358489990234, -19.132795333862305, 14.613641738891602, 10.72927474975586, -9.848973274230957, -10.373320579528809, 25.420923233032227, -27.655759811401367, 12.544028282165527, -22.151031494140625, 21.636869430541992, -12.041404724121094, -2.823726177215576, 11.871167182922363, 13.522286415100098, -30.532197952270508, 9.989394187927246], [25.697702407836914, -4.000048637390137, 9.681816101074219, -17.385761260986328, 9.321561813354492, -8.894309043884277, 11.679731369018555, -1.5326552391052246, 16.831012725830078, -11.139840126037598, 13.34061050415039, 5.904995918273926, 2.0028367042541504, 32.29840087890625, 2.0738279819488525, 24.878263473510742, 39.54322814941406, -25.815082550048828, -9.245344161987305, -2.264810085296631, 5.083963394165039, 27.933753967285156, 7.012272834777832, -9.4677152633667, 11.011678695678711, 8.56439208984375, -32.63774108886719, 29.94744873046875, -10.464282035827637, -14.718184471130371, -3.4877867698669434, 13.727158546447754, 17.592979431152344, -6.7516560554504395, -10.799667358398438, 18.140575408935547, -20.49847412109375, -2.8167364597320557, -12.933042526245117, 20.600038528442383, 0.8110042214393616, 12.915342330932617, 1.7326477766036987, 10.908315658569336, 16.348363876342773, -27.160823822021484, 17.780332565307617, 38.23886489868164, 21.761009216308594, -4.87703800201416, -5.751925945281982, -6.474628448486328, -3.287850856781006, 34.31264877319336, 12.626666069030762, -25.308547973632812, 10.299979209899902, 10.863040924072266, -21.546836853027344, -5.181360721588135, 34.933837890625, -4.535680294036865, 11.512367248535156, -12.32708740234375]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.45740556716918945, 0.930532693862915, -0.10832682996988297, -2.580386161804199, 0.07252494990825653, 1.427647590637207, 0.6679513454437256, 0.7790650129318237, 0.3281821012496948, 1.2888925075531006, -0.5779718160629272, -0.8435995578765869, -0.2990589737892151, 1.9545849561691284, -0.010610024444758892, 0.5316979289054871, -0.9052947759628296, 0.0546787865459919, 2.3046727180480957, 0.190452441573143, 0.4072955250740051, -0.5829744338989258, -1.9223777055740356, -1.1097267866134644, -0.1906735897064209, -0.07835248857736588, -1.1611652374267578, -1.4099986553192139, -0.8841407895088196, -0.07073371857404709, 1.1780105829238892, -0.5173557996749878, -0.32592931389808655, 0.16636212170124054, -0.01886589638888836, 1.7314904928207397, -0.23086988925933838, 0.1594640165567398, 0.5602381825447083, -0.010935623198747635, -0.4856100380420685, 0.4818049967288971, 1.3663983345031738, 0.3268226981163025, -0.8064166903495789, 1.294379711151123, 0.2867898941040039, -0.8438195586204529, -0.3181487023830414, -0.43845778703689575, 0.16385452449321747, -0.49401286244392395, -1.094993233680725, 1.1865925788879395, -1.467175841331482, 0.6395779252052307, -0.6776178479194641, -0.7416632771492004, 0.04626733437180519, -0.3991393744945526, 1.3375128507614136, -0.9454372525215149, -0.6355392336845398, -0.5015223622322083, 0.22733326256275177, -1.4841228723526, 1.7677487134933472, 0.20719407498836517, -0.2016986608505249, -0.9551469087600708, -0.9897408485412598, 1.7490493059158325, -1.5890758037567139, -0.1352853626012802, 0.5765219330787659, -2.1438493728637695, 2.0035548210144043, 0.16491751372814178, 0.6457077264785767, 0.04981305077672005, 0.7379991412162781, -0.9894128441810608, 1.7713489532470703, -2.220951557159424, -0.533719003200531, -0.1803768426179886, 0.19659492373466492, 2.169191598892212, 0.12984655797481537, -1.3879804611206055, 0.8844531178474426, 0.1992998868227005, 1.169019341468811, 1.2132359743118286, -0.5796703100204468, -2.0391910076141357, 1.250227689743042, -0.6982977986335754, -2.086268901824951, -0.5328786969184875, 0.5242679119110107, -0.4826315939426422, -0.20282116532325745, -0.9190108776092529, 0.9217941761016846, -0.8445487022399902, -0.10101047158241272, 0.19688114523887634, -0.4147321879863739, 0.614412784576416, -0.5250137448310852, -0.09274157881736755, -1.5254356861114502, -0.2911069095134735, 0.5933833718299866, 0.9407404661178589, 1.0236074924468994, -1.1243451833724976, 0.16208839416503906, 0.12558574974536896, 0.11368796974420547, -0.2586316764354706, -1.8805155754089355, -0.2804326117038727, -0.07662016153335571, -2.225951671600342, -0.16510841250419617, 0.6928834915161133, 0.5288716554641724, -0.47045907378196716, 0.9127317070960999, -0.5648886561393738, -1.1089578866958618, 0.08594716340303421, 0.8790185451507568, -0.589333713054657, 0.9959307909011841, 0.2483345866203308, 1.7943065166473389, 0.20065483450889587, -0.43491309881210327, -0.5391262173652649, -0.007821120321750641, 0.7840198874473572, -0.5661625862121582, -2.26225209236145, 1.9388625621795654, -0.529843807220459, -1.3945170640945435, 2.133211851119995, 0.329226553440094, -1.3924338817596436, 0.3328886926174164, -2.6066768169403076, -0.8360490798950195, 0.2095431536436081, 1.3294892311096191, 0.010931702330708504, 0.4676399230957031, 1.0519473552703857, -0.03428292274475098, 0.6511952877044678, 1.6928212642669678, 1.1132481098175049, 0.6685128808021545, -0.6744322776794434, -0.690910816192627, -1.556213617324829, 0.3519447445869446, 1.0950360298156738, -0.5816178321838379, 0.4152299463748932, -1.6725250482559204, -0.17323580384254456, -0.5981267690658569, 0.8293150663375854, -1.688870906829834, 0.7377490997314453, 0.16566066443920135, -0.34419503808021545, -0.07753278315067291, -1.7650562524795532, -0.2646237015724182, 0.1954493671655655, 0.09956339001655579, -0.5129477977752686, 0.8540624380111694, 0.05821133032441139, 0.11792533844709396, 1.0199358463287354, 2.4703052043914795, 0.5788525342941284, -0.9447476863861084, 0.022135432809591293, -0.3833440840244293, -0.9716337323188782, 0.9231290817260742, 1.4756196737289429, -0.03214206174015999, -0.8208069801330566, 1.1198513507843018, -1.0339158773422241, -1.2609416246414185, -0.8956738710403442, -0.4884500801563263, 0.1669728010892868, -0.5651568174362183, -0.881554365158081, 0.22932255268096924, -2.0451252460479736, -0.5982277989387512, -1.5817598104476929, -0.020589323714375496, 0.43428972363471985, 2.048574209213257, -1.48617684841156, 0.8561290502548218, 1.3590353727340698, 0.5073907971382141, 0.5016379356384277, 0.11747200787067413, -1.0131453275680542, 1.1128045320510864, -2.33799147605896, -1.7817238569259644, 0.3758954405784607, 0.10039771348237991, -0.5989188551902771, 0.8301036953926086, -1.716560959815979, 0.4591795802116394, -1.1190917491912842, -0.5881745219230652, 1.0926145315170288, -0.3384018838405609, 0.6011117696762085, 1.1757590770721436, 0.3548766076564789, -0.48853421211242676, -1.8545324802398682, -0.8519524931907654, -0.5317907333374023, 0.44655296206474304, -1.0264527797698975, -0.6137362122535706, -1.2121641635894775, 0.06768575310707092, 0.8055439591407776, -0.18996645510196686, 0.0178974661976099, -0.8032963871955872, 0.6119652986526489, -1.2811676263809204, -1.590885877609253, 0.890145480632782, 0.5256056785583496, 2.0947203636169434, -0.889049768447876, 1.5780627727508545, -0.08304011076688766, -0.5447955131530762, 0.40398889780044556, 0.09758690744638443, -0.349263459444046, 0.20091639459133148, 0.8225954174995422, -0.08146662265062332, -0.5531905889511108, 2.3063948154449463, 0.6892852783203125, -0.25299882888793945, -0.5814422965049744, 0.022878188639879227, 0.5754445195198059, 0.1035672277212143, -0.30259689688682556, -1.35396146774292, 0.42034828662872314, 0.22353652119636536, -1.4314450025558472, 0.5052680373191833, 0.27307289838790894, -0.7993295192718506, 1.6183109283447266, -0.7339926958084106, 1.079158067703247, 1.1454453468322754, -1.364478588104248, 0.8593260645866394, 0.36451196670532227, 0.2233867049217224, 1.510182499885559, 0.806209921836853, 0.23097456991672516, -0.916678786277771, 1.1233313083648682, 1.1761873960494995, -0.18311314284801483, 1.5917528867721558, 0.08675561845302582], [1.9045329093933105, 1.5680904388427734, -0.061079759150743484, -0.7458871603012085, 0.09117864817380905, 0.4536600410938263, -1.581667423248291, 0.3549417555332184, -0.23745569586753845, -1.2137010097503662, 0.41722655296325684, -1.0541518926620483, -0.15350671112537384, 0.6456754803657532, 0.2737103998661041, 0.6036940813064575, -0.3602130115032196, -0.13680841028690338, -0.9043282270431519, -0.34137916564941406, -2.1271774768829346, -0.5300881266593933, -0.14653003215789795, -0.3349582552909851, -1.5149527788162231, 0.8277300000190735, -0.10003811866044998, -1.3305373191833496, -1.5541017055511475, 0.6101391911506653, 0.8652373552322388, -0.6457926630973816, -1.0378503799438477, -2.4509453773498535, 0.40978094935417175, -0.37879714369773865, -0.03664706274867058, 0.12979158759117126, -1.0573837757110596, -0.23854635655879974, 0.20154766738414764, 0.9178938269615173, -0.4169677197933197, -0.4667818248271942, 0.07665964961051941, 0.9985226988792419, 0.8248347640037537, -0.14359942078590393, 1.32015860080719, -1.091506838798523, 1.9976919889450073, -0.949734091758728, -0.46949970722198486, 0.9158130288124084, 0.2748953700065613, 1.3588284254074097, -1.0944762229919434, 1.6192952394485474, -0.2561674118041992, -0.056675925850868225, -2.167377233505249, 2.0835824012756348, 0.630486249923706, -2.0116446018218994, 0.2604736387729645, 1.1940150260925293, -0.20713604986667633, 1.1246832609176636, 0.9524435997009277, 0.7100029587745667, 0.1835700124502182, 1.253121018409729, 0.3014174997806549, -1.0684958696365356, 0.9973467588424683, -0.5978184342384338, -0.049706749618053436, 2.040926933288574, 2.0312724113464355, 0.6746267080307007, -0.39903590083122253, -0.9064560532569885, -0.3228757977485657, -1.5338537693023682, 0.08585060387849808, 0.7321326732635498, -0.7264837622642517, 0.5009825825691223, 1.2456988096237183, 1.4052855968475342, -0.27178892493247986, -1.5007905960083008, 0.6455398201942444, 1.2094372510910034, -0.24773317575454712, -1.5326688289642334, 0.5244415402412415, -0.43794652819633484, -1.938752293586731, -0.4804380238056183, -1.017823338508606, -0.4523770213127136, 1.3308312892913818, -0.038397371768951416, -0.10471300035715103, 1.4490641355514526, -1.9691672325134277, 0.7226520776748657, 0.35046321153640747, 0.006634265184402466, -0.2135377824306488, 0.953890860080719, -1.1356773376464844, 1.390130877494812, 0.5322605967521667, -1.4361917972564697, 0.500403881072998, -0.2716031074523926, 0.7784446477890015, -0.02096783183515072, -0.2831583023071289, 0.16592217981815338, 0.04387567192316055, -0.3823778033256531, -0.469502717256546, -0.589150607585907, -1.4427059888839722, 0.5112425088882446, 0.7799947261810303, -0.7099920511245728, 0.3589050769805908, 0.3342411518096924, -0.6398493051528931, -0.6778099536895752, -0.31398114562034607, -0.15203821659088135, 1.007519006729126, 0.581562340259552, 1.1006187200546265, 0.7202408313751221, 0.6358383297920227, 0.40377309918403625, 0.5392642617225647, 0.9125828146934509, 0.3858012855052948, -1.9831496477127075, 0.5128507614135742, 0.3528710901737213, 0.9399511814117432, 1.2674896717071533, -0.41845422983169556, 0.6429726481437683, -0.17544646561145782, -0.2115641087293625, -0.23330336809158325, 2.220510244369507, 1.5142035484313965, 1.5927293300628662, -0.775521457195282, -0.7493284940719604, 0.25471413135528564, -0.6098750233650208, -0.8209599256515503, 1.4464390277862549, -0.5977026224136353, -0.398979514837265, -0.3699246644973755, -0.4599396288394928, 0.5259411931037903, -1.7940484285354614, 0.5569134950637817, 0.31236863136291504, -0.3891857862472534, 0.37813878059387207, 0.06983811408281326, 0.20149709284305573, 0.7784177660942078, -0.5374443531036377, 0.9045708775520325, -0.13653799891471863, -0.3018797039985657, -0.03477349504828453, 0.15991832315921783, -0.2568367123603821, -0.7215574383735657, 1.500937819480896, -1.3044768571853638, -0.5652062892913818, -1.4537544250488281, 0.9189916253089905, 1.0295506715774536, 0.05179150030016899, -0.4741669297218323, 0.5880916118621826, -1.2775545120239258, 0.07807578891515732, -1.9267299175262451, 0.2365492433309555, 0.0692892074584961, 0.07724817842245102, -0.18753425776958466, 0.9216510653495789, -1.9766132831573486, 0.6419063210487366, -0.1481596976518631, -0.6594754457473755, -0.07999390363693237, -0.3406662344932556, -1.00163733959198, -0.2785889506340027, 0.20530863106250763, -0.3045145273208618, 1.2093894481658936, -0.16842901706695557, 0.8329272270202637, 1.6469699144363403, 2.60701584815979, -0.781735897064209, 0.19723759591579437, 0.5159456133842468, 1.4892107248306274, -0.03524782136082649, 0.7304878234863281, -0.7481955289840698, -0.35371798276901245, 0.8214961886405945, 2.3404276371002197, -0.9475030303001404, 0.1441669762134552, 0.9296042323112488, 1.1131227016448975, -0.9182777404785156, 0.75154048204422, 0.18399086594581604, 2.5991313457489014, 0.6748217940330505, 0.6738327741622925, 1.0714168548583984, -1.0668003559112549, -1.383104920387268, 0.5599263310432434, -1.8246885538101196, 0.011523396708071232, -0.6041253805160522, -0.31491023302078247, -0.14633990824222565, 0.7292100191116333, -0.5616429448127747, -0.9444974064826965, 0.7949712872505188, 0.29026392102241516, 0.8320772647857666, 0.6143544316291809, -0.46199312806129456, 1.2312101125717163, -0.517866313457489, 0.4254503548145294, 1.4994829893112183, 0.5698542594909668, 0.207980215549469, 1.7043888568878174, -0.11796531826257706, -0.3853702247142792, 0.9486789107322693, 0.1504381000995636, -0.4292404055595398, 1.1669301986694336, -0.051361773163080215, 0.05134044960141182, 0.23788124322891235, 0.5202317237854004, -0.022570323199033737, -0.47410327196121216, 0.15310624241828918, 0.8243685960769653, 1.3344637155532837, -0.3780677914619446, -1.4559873342514038, 0.31404203176498413, -0.9935374855995178, -0.20227137207984924, -0.7223720550537109, 0.5572015643119812, -2.10294771194458, 1.357527494430542, -0.250677227973938, 1.3828498125076294, 0.5751749277114868, -0.24903468787670135, -0.26705291867256165, -0.28740695118904114, -0.7172833681106567, -0.6840263605117798, -1.866563320159912, -0.24689245223999023, -0.7530432343482971, 0.29975342750549316, -1.216787576675415, 0.18075689673423767, -0.47555437684059143], [0.7723739147186279, -0.7954173684120178, 0.1934414505958557, -2.6458353996276855, 1.1179903745651245, 1.6291108131408691, 3.567606210708618, -0.6459656953811646, 0.2733627259731293, -0.42823681235313416, -0.6368885040283203, -0.7999550700187683, 0.1596875637769699, 1.0654726028442383, -0.21538139879703522, 1.0489099025726318, -0.03426443785429001, 0.38536131381988525, -0.2870583236217499, -0.1433301717042923, 1.3758870363235474, -1.4952304363250732, -1.4853323698043823, 0.2527178227901459, 1.064131736755371, 0.030186939984560013, -1.0351004600524902, -0.4639737904071808, -0.6047459244728088, 0.318198025226593, -0.006357905920594931, 0.09899598360061646, 0.08837980777025223, 0.6372337341308594, 0.5844245553016663, 0.9455164670944214, -1.1561816930770874, -0.02783997543156147, -2.028002977371216, 0.07926119118928909, -0.03666255250573158, 0.04668460413813591, 0.20226077735424042, 0.034079909324645996, 1.7125710248947144, 0.586139976978302, -0.4456343352794647, 1.4965453147888184, 1.1304527521133423, 1.0683612823486328, -1.2517088651657104, 0.5659748315811157, -1.245383858680725, 0.7921762466430664, -3.0687568187713623, -1.1367841958999634, 0.2214316576719284, 0.7108340859413147, -0.506756603717804, -0.45679542422294617, 0.016628989949822426, -0.2670682668685913, -0.3330666124820709, 0.6996307373046875, -0.03394877910614014, -1.5065141916275024, -0.6761821508407593, -1.7956695556640625, -1.9013497829437256, 0.13345910608768463, -0.8601298928260803, 0.7130847573280334, 0.874255895614624, -0.9997168779373169, -2.2779574394226074, -1.249032974243164, 0.23150834441184998, 0.6497553586959839, 1.6007136106491089, -0.17621126770973206, -0.6385726928710938, -0.380687415599823, -0.870477020740509, -0.9650147557258606, -0.5508012771606445, 0.7766087055206299, -1.8004748821258545, 0.022345032542943954, -1.4009565114974976, -1.6132733821868896, -0.45395541191101074, -0.4386903941631317, 0.799782395362854, 0.5813964009284973, -0.531801700592041, 0.6107600927352905, 0.09725628048181534, 2.746927499771118, -1.4373003244400024, -0.34337911009788513, -0.8096948266029358, -3.0982494354248047, -0.5524654388427734, 0.3219135105609894, -0.6368176937103271, -0.23475271463394165, 0.41963881254196167, -1.2323905229568481, -0.6019998788833618, 0.22618749737739563, 0.0868946984410286, 0.88864666223526, 0.33094626665115356, 0.8818508386611938, -1.4813978672027588, 0.3060804307460785, -0.20866729319095612, -2.0169897079467773, -0.2781008780002594, 0.9349426031112671, -1.2119001150131226, 0.6005169749259949, -0.07495596259832382, -0.6775460243225098, -0.5276197791099548, -0.019816165789961815, -0.6986815333366394, -1.1408638954162598, 1.5948835611343384, -2.409813165664673, 0.8551815152168274, 1.067732572555542, -1.3289484977722168, -0.28196749091148376, -0.8088108897209167, -0.5101791024208069, -0.8241204619407654, 0.1800505816936493, 0.13464988768100739, 0.8374557495117188, -2.2185137271881104, -0.011822935193777084, 1.478096842765808, -0.5316059589385986, 1.2993340492248535, 0.008009709417819977, -1.0441617965698242, -0.8794173002243042, 0.1601988971233368, -2.248138666152954, -1.3823926448822021, 2.8353610038757324, -0.3332868218421936, -0.5412718057632446, 0.3987904191017151, -0.04853522405028343, 1.2824339866638184, 0.6701768636703491, 2.107959747314453, 0.6653145551681519, -1.2107700109481812, -1.175636887550354, 1.6205421686172485, 0.6933963894844055, -1.2319973707199097, -0.4433930218219757, -0.007280388846993446, 0.47977906465530396, -0.9929550290107727, -0.9578800797462463, -0.7039284706115723, 0.7177836894989014, -0.3601519465446472, 0.8147594332695007, 0.2551564574241638, 1.735589623451233, -0.6235249042510986, -0.9606356620788574, -0.6520758271217346, -0.32838812470436096, -0.6043472290039062, 0.8266494870185852, 0.9113286137580872, 0.6623174548149109, -0.10456906259059906, -1.0326803922653198, -0.08003050833940506, 0.4243287444114685, 0.2669786810874939, 0.4322054982185364, -0.8889650702476501, -0.956605851650238, 0.8906641006469727, -0.2815757989883423, -0.7571665048599243, 0.3733741044998169, 0.32016289234161377, -1.9915976524353027, -1.4896602630615234, 1.3081328868865967, -1.9541223049163818, -1.7428733110427856, -1.0675595998764038, 1.2984172105789185, -1.0891566276550293, -0.024873925372958183, -0.03566192835569382, 0.3877916634082794, -0.4375957250595093, 0.35048946738243103, 2.1844139099121094, -1.5123646259307861, -0.559418797492981, -1.2153395414352417, 0.006765125785022974, -0.5251017808914185, 0.5408262610435486, -0.00019383776816539466, 0.2658762037754059, -0.6194333434104919, -1.4213786125183105, 0.7230794429779053, 1.5147961378097534, 1.445298433303833, -1.7368911504745483, 1.2441736459732056, 1.3230721950531006, 1.0879709720611572, 1.2395628690719604, -0.5941352844238281, -0.5879392027854919, -0.29744961857795715, 0.48543187975883484, -0.5192328691482544, -0.08278332650661469, 0.49376869201660156, -0.49318742752075195, -0.12091871351003647, 0.3365978002548218, 0.12241443991661072, 0.7641406059265137, -1.0760562419891357, 0.260137677192688, 0.9679886698722839, 2.3389997482299805, 0.16781297326087952, -0.9743627309799194, -0.37378260493278503, 0.2970741391181946, 1.042627215385437, 1.437769889831543, -1.2825710773468018, 0.9128144979476929, 1.7190247774124146, 1.4652894735336304, 0.6229905486106873, -0.23659320175647736, -0.48031917214393616, 1.364151954650879, 0.7084193229675293, 0.8277656435966492, -1.03044593334198, -0.5849958658218384, -0.6135229468345642, 0.1907467395067215, 1.6747256517410278, 0.8864327073097229, 1.3272784948349, -0.11642571538686752, 0.044205307960510254, -0.9789875149726868, -0.5368396043777466, -0.3937280774116516, 0.8883604407310486, -0.27719926834106445, 0.2844158113002777, 0.3277741074562073, 1.0840692520141602, 0.5265689492225647, -0.8073874115943909, -0.33661073446273804, 0.5436524748802185, -0.0008818862261250615, -0.8274150490760803, -0.07059749960899353, -2.463700532913208, 0.3231363296508789, 1.8811842203140259, 0.6310366988182068, 0.5424607992172241, 0.7548181414604187, 1.0378228425979614, -1.047207236289978, 1.4488614797592163, -0.029995569959282875, 1.4337077140808105, 0.7637256383895874, 0.8020576238632202, 0.5454918146133423, -0.3381927013397217], [-0.7757109999656677, -0.25385716557502747, 1.0074234008789062, -1.1483299732208252, -0.8066431879997253, 0.015363769605755806, -1.0962218046188354, -1.4238519668579102, 0.8126661777496338, 0.5828817486763, 0.5624586939811707, 0.3639993667602539, -1.478500247001648, -0.4590037167072296, -0.6248002052307129, 1.751159906387329, -1.052981972694397, 0.44981417059898376, 0.27433037757873535, 0.1611076146364212, -0.3256937861442566, 0.030353529378771782, 0.19207730889320374, 0.015553552657365799, 0.47727495431900024, 0.2587863802909851, -2.0699219703674316, -0.06201617047190666, 1.2202006578445435, -1.4432029724121094, -1.1292988061904907, 0.7237730622291565, 0.42540210485458374, 0.6860551238059998, -0.7841303944587708, -0.811809241771698, 0.19121959805488586, -1.4729057550430298, 1.2728158235549927, -0.4169129729270935, 1.2888545989990234, 0.39128443598747253, 0.9392488598823547, -0.6414595246315002, 1.5964279174804688, -0.5430533289909363, 0.07142727077007294, 0.20569735765457153, 0.47132954001426697, 0.8858304023742676, -0.21339087188243866, -1.519506812095642, -1.170615792274475, -0.6195946335792542, -0.6412086486816406, 1.4655704498291016, -0.22405685484409332, -0.055290572345256805, -0.8871259093284607, -0.1949433535337448, -1.2917400598526, -0.625337541103363, -0.673991322517395, 1.081742525100708, -0.21910297870635986, 0.4971844553947449, 0.002363555831834674, 0.7081097960472107, -0.8508034348487854, -0.6897506713867188, 0.7733607888221741, -0.6959336996078491, -1.989047884941101, -1.353200078010559, 0.5224183201789856, 0.9065257906913757, -0.48837047815322876, -1.4363535642623901, -0.3537076711654663, 0.6660917401313782, -1.190190315246582, 0.3557490408420563, -0.5374584197998047, 0.6131356358528137, -0.35661059617996216, 0.032897982746362686, 0.5531679391860962, 0.54253751039505, -0.4204995930194855, -0.9566745758056641, 0.5184445977210999, 0.14281462132930756, -0.8960563540458679, 0.6002073884010315, 0.37987643480300903, 2.112847328186035, -1.0275352001190186, 0.7725721597671509, -0.5593204498291016, 0.17304565012454987, 0.8658474683761597, 0.6860162615776062, 0.1236504316329956, 0.12416880577802658, -2.34501051902771, 0.5209027528762817, 0.41615214943885803, 0.36662307381629944, -0.7877711653709412, -1.218053936958313, -1.4890835285186768, -0.022871317341923714, -0.5908856987953186, -0.43180444836616516, -0.3014600872993469, 0.4945972263813019, 0.38031789660453796, -1.4920318126678467, 0.5383044481277466, -1.7813761234283447, 0.855595052242279, -0.7845743298530579, 0.9435887932777405, 1.1717257499694824, 0.15280860662460327, 0.3600928783416748, -1.1203269958496094, 2.2563636302948, -1.5997527837753296, -0.7114996314048767, 0.5203858613967896, -0.5637025833129883, -0.012129513546824455, 0.36051833629608154, -0.5142245292663574, 0.8765862584114075, -0.0023402106016874313, -0.4447673559188843, -0.6248416304588318, 0.34713274240493774, -0.6216087341308594, 0.6920059323310852, -0.09731505811214447, -2.2289493083953857, 0.3043166399002075, 1.7588127851486206, 0.5195289254188538, 0.8724580407142639, 1.529338002204895, 0.6625204682350159, -1.325468897819519, -0.4467887282371521, 0.4104517996311188, -1.3985776901245117, 0.9743989109992981, -1.1047372817993164, -2.4180262088775635, -0.3771789073944092, -1.3003311157226562, -0.42658504843711853, 1.5744810104370117, -0.30284491181373596, -0.262794554233551, 1.2760725021362305, -1.4804846048355103, 1.1778523921966553, -1.811442494392395, 0.1212049275636673, -1.1466612815856934, -0.12817144393920898, 0.8659244179725647, -0.5254542827606201, 1.0688636302947998, 1.2467173337936401, 0.028540756553411484, 0.8490369915962219, 0.6732736229896545, -0.9451136589050293, -1.971367597579956, -0.475553423166275, 0.04744177684187889, -0.4536035656929016, -0.788375973701477, -0.8816037774085999, 1.057108998298645, -1.7330378293991089, 0.23323731124401093, 0.5674434900283813, 0.3646056354045868, -0.06887658685445786, -0.7852967977523804, 0.2645062804222107, -0.9011700749397278, -1.3284555673599243, -0.42463845014572144, 0.22586053609848022, -0.6261600255966187, -1.4969826936721802, 1.9674828052520752, 1.6041028499603271, -0.45291900634765625, -0.6854546070098877, -0.3258223831653595, -0.4460277259349823, 0.6002416610717773, -0.4771912693977356, 1.9204412698745728, 1.3152778148651123, 0.6819262504577637, 1.395355224609375, -0.5032666325569153, -1.3005269765853882, 0.15019847452640533, -0.7131149172782898, -0.9739093780517578, -1.9964765310287476, 0.7815031409263611, -0.6416416168212891, -1.4264445304870605, -1.7110726833343506, -0.32354792952537537, 1.3219751119613647, -1.0743381977081299, 1.3529115915298462, -0.6119585037231445, -0.12577329576015472, -0.5573963522911072, 0.3926701247692108, 0.8306103944778442, -1.4155097007751465, 0.8216283321380615, 1.2130118608474731, 0.6035585999488831, -0.6818968057632446, -0.8133214712142944, -0.04194334149360657, 0.19292695820331573, 0.1866370588541031, -0.3444409668445587, 0.5624539852142334, -1.1788876056671143, 0.4586019814014435, 2.2186825275421143, 0.5353482961654663, -0.30054357647895813, 1.0922744274139404, -2.4658117294311523, 0.08194366097450256, 0.6041498780250549, -1.237091064453125, 0.010589390993118286, -0.67535799741745, 2.329113721847534, -0.7182420492172241, -0.6544985175132751, 0.16072678565979004, -0.5414638519287109, 0.2686121463775635, -0.41359183192253113, -1.7713922262191772, -1.0881505012512207, 0.13974101841449738, -0.9267237186431885, 0.37214818596839905, -0.3052963316440582, 0.3799152076244354, -1.2605838775634766, -1.0103626251220703, 0.2502926290035248, -1.0693310499191284, -1.061502456665039, 0.8940097093582153, 1.8265128135681152, 1.1586065292358398, -1.3849818706512451, -1.1632411479949951, -1.3616445064544678, 0.7179068326950073, -0.757863461971283, 0.3722931146621704, -0.029101060703396797, -0.362901896238327, -2.1598193645477295, -0.3913296163082123, -0.4955793619155884, -1.9694268703460693, -0.30513525009155273, -1.4329155683517456, -0.6337652206420898, -0.5962650775909424, 2.137096643447876, -0.02385643869638443, 0.19445756077766418, -1.4615901708602905, 0.2601622939109802, 2.7152750492095947, 0.12075889855623245, 0.6083499789237976, -0.4784320890903473, 1.1337611675262451], [-1.1692116260528564, 0.7347186803817749, 0.43372493982315063, -0.3079477846622467, -1.152305245399475, -0.9545409679412842, -1.5951426029205322, 0.014340651221573353, 1.6166393756866455, -0.8379101753234863, 0.20180180668830872, 1.040447473526001, 0.22406411170959473, -0.5858976244926453, 0.12552319467067719, 0.14165470004081726, -1.5647135972976685, -1.3189736604690552, -0.18627797067165375, 1.5742478370666504, 0.3349403142929077, 0.4294421970844269, 1.0635249614715576, 1.2468129396438599, 0.38579487800598145, 0.7475022077560425, -0.04809025675058365, -0.147457093000412, -1.0558897256851196, -0.13944803178310394, -0.19549278914928436, 0.7036113142967224, -1.1817814111709595, 0.27430039644241333, 0.6262058019638062, 0.42937934398651123, -0.35224106907844543, -0.6235698461532593, 0.060655415058135986, -0.18834909796714783, -0.10223760455846786, 0.11253507435321808, 0.808332622051239, -1.1085056066513062, 0.9206516146659851, 1.1952589750289917, 1.091242790222168, 1.1941149234771729, -0.42169564962387085, -0.08278174698352814, 0.6685291528701782, 0.03855461999773979, 0.049864377826452255, 0.07226160168647766, -0.757851243019104, -0.08986611664295197, -0.4718397259712219, 0.3230571746826172, 2.1215670108795166, 0.6419224739074707, -0.008907500654459, -1.2503337860107422, 0.5819883942604065, 1.391036868095398, -0.2896723747253418, -0.539353609085083, -0.2678648829460144, 0.6522237062454224, 1.3626781702041626, -0.34419751167297363, 0.3714621067047119, -0.2334630787372589, 0.1688421368598938, 2.6326725482940674, 0.37919431924819946, 0.663033664226532, -0.6491995453834534, -2.0800867080688477, -0.04798376187682152, -0.5382110476493835, -0.26905015110969543, -1.1537091732025146, 0.2938556671142578, 1.2002240419387817, -1.0587552785873413, 1.4421019554138184, 1.0060876607894897, -0.3054695725440979, 1.2676286697387695, -0.452783465385437, -1.5919978618621826, -0.5339898467063904, 0.7643535733222961, 1.3001042604446411, 0.46034279465675354, 0.1570836901664734, -0.22100785374641418, -0.09104928374290466, 0.17687568068504333, 0.65060955286026, 0.41377002000808716, 0.019656162708997726, 0.4800679087638855, -0.9028127193450928, 0.7963162660598755, 0.0550357811152935, -0.2779539227485657, 1.7414278984069824, -1.0901849269866943, 1.6168012619018555, 0.06395578384399414, -0.14103879034519196, -0.7328693270683289, -0.1718752533197403, -1.1705600023269653, -0.48416951298713684, -1.4112745523452759, -0.3277972340583801, 0.9437373876571655, 0.5091285109519958, -2.084841012954712, -0.4387710988521576, -0.8398252129554749, 1.1294249296188354, -2.033581495285034, 1.7632571458816528, 0.022376038134098053, -0.012588877230882645, 0.7327002882957458, 0.4102579653263092, -0.1488492488861084, -0.1241479367017746, -1.1080740690231323, 0.31439465284347534, 0.38978639245033264, -0.6607708930969238, 1.5826436281204224, -0.6537350416183472, -1.341389536857605, 0.23833057284355164, -0.8312327861785889, 0.7552387714385986, 0.046125590801239014, -0.4197624921798706, -1.043835163116455, -1.0834726095199585, -0.27185457944869995, 0.8657106757164001, 0.20701591670513153, -2.700347423553467, -0.38993173837661743, 1.1257858276367188, 0.2620479464530945, -1.5373531579971313, 0.49897611141204834, -1.5050960779190063, 1.8050153255462646, 1.1864739656448364, 1.0825234651565552, 1.4431387186050415, -1.2400485277175903, -1.4828522205352783, 0.798221230506897, -0.7712995409965515, -0.15807227790355682, 0.11052165180444717, -0.2618589997291565, 0.0046594031155109406, -0.6707383394241333, -0.41413646936416626, 0.8302308320999146, -0.7906599044799805, -1.3270829916000366, -0.8625437021255493, -1.3772127628326416, 0.5616329312324524, 0.37652039527893066, 0.29328081011772156, 0.15888603031635284, 0.13039572536945343, -0.41580191254615784, -1.6282941102981567, 0.37058839201927185, -0.38747453689575195, 1.1493531465530396, -0.776160717010498, 0.013466733507812023, 1.9563151597976685, 0.08621109277009964, 0.65776127576828, -0.36774909496307373, -1.4428454637527466, 1.5047893524169922, -0.16430892050266266, -1.2799495458602905, -1.1836899518966675, -0.33803901076316833, -0.37325066328048706, 0.27074745297431946, 1.3951548337936401, 0.3290976285934448, 0.4771673381328583, 1.0181576013565063, -0.3092542588710785, 2.247073173522949, 0.8987704515457153, 0.7629384994506836, -0.1481626331806183, 0.035752467811107635, -0.15229161083698273, 0.6004400253295898, -1.0133787393569946, 0.5669308304786682, -1.0657007694244385, -0.49917998909950256, 0.4929453134536743, 0.28096017241477966, 0.6042869687080383, -0.7003760933876038, 0.01884293183684349, -0.8983514308929443, 0.8416286110877991, 0.6737128496170044, -2.139911413192749, -1.065606713294983, 0.8980906009674072, -0.10704322904348373, -0.23650477826595306, 0.4862241744995117, -0.48291587829589844, -0.06500190496444702, 0.4422511160373688, 0.219538614153862, -0.8312410116195679, 0.44081440567970276, 0.3025476634502411, -1.2884126901626587, -0.8565778136253357, 0.5638513565063477, -0.32166460156440735, 0.8472801446914673, -0.060403257608413696, -0.30336233973503113, 1.125646710395813, 0.9622864127159119, 0.1557433009147644, 0.6535268425941467, 0.9472436904907227, -0.18640047311782837, -0.0646631196141243, -0.19319887459278107, 0.5129512548446655, 0.6959456205368042, -2.683321237564087, 0.01379565242677927, 0.42337656021118164, -0.20332980155944824, -0.672713041305542, 1.3906623125076294, -0.8689333200454712, 1.0494575500488281, -2.0646560192108154, 0.9446187019348145, -0.3939512372016907, 0.5189365744590759, 0.5183538198471069, 0.42673957347869873, 0.11549393832683563, -0.030204344540834427, -1.3564774990081787, -0.5200108885765076, -1.6662248373031616, -0.8814889788627625, -0.25822538137435913, -1.2443195581436157, 1.7656441926956177, 1.0321507453918457, 1.2645732164382935, -0.689860999584198, -0.641696035861969, 2.0491302013397217, 0.19941619038581848, 1.5004806518554688, 1.1232870817184448, 1.4418869018554688, -0.4316209852695465, -1.0883305072784424, 0.06103940308094025, -1.2499843835830688, 0.35794368386268616, -2.0401434898376465, -0.2612442672252655, 0.02340560592710972, -0.3585098683834076, -0.08298126608133316, -0.15903593599796295, 0.8696095943450928, -0.6675912141799927, 0.9780938625335693, -0.5681196451187134], [1.6787405014038086, 0.1748426854610443, 0.2950535714626312, -0.8412044048309326, 2.437204360961914, -2.616328239440918, -0.24582737684249878, -0.6412865519523621, 1.3111155033111572, -0.03118923492729664, 0.19176751375198364, -1.5095621347427368, 0.1119862049818039, -1.21929931640625, 1.1475237607955933, 0.4488917589187622, -0.21063107252120972, -0.6338016986846924, 1.9151427745819092, 0.346613347530365, 0.8388010263442993, 2.0526933670043945, -0.34788820147514343, 0.37113288044929504, -1.1935014724731445, -0.14153330028057098, -1.3147389888763428, 1.1219947338104248, 0.549858033657074, 0.7819761037826538, -1.298439860343933, -0.9541923999786377, 1.0589179992675781, -1.1392995119094849, 0.11148155480623245, 0.34008529782295227, 0.541513979434967, 0.35746708512306213, -0.5804188251495361, 1.3486741781234741, -0.47775354981422424, 0.6496348977088928, -0.5620583891868591, -0.4272219240665436, -0.12349426746368408, 0.9895902872085571, 1.748708963394165, 1.540077805519104, -0.6391456127166748, -0.11359217017889023, 1.4580445289611816, 0.8499978184700012, 0.3690425455570221, -0.32725411653518677, -1.2661571502685547, -0.44248124957084656, 0.7368230223655701, 1.509340763092041, 0.7451713681221008, 0.9637941122055054, -0.7858623266220093, 2.492734432220459, 0.021869022399187088, 0.1633950024843216, -0.3765053451061249, -1.5365856885910034, 1.076146125793457, 1.2609516382217407, -1.5714677572250366, 0.6911975145339966, -0.07944435626268387, 1.2573531866073608, -1.7133561372756958, -1.504875659942627, 0.14060892164707184, 0.37225770950317383, -0.1538613736629486, 0.12855671346187592, -1.275680661201477, -0.644899845123291, 0.027318643406033516, -0.2720589339733124, -0.12760484218597412, -1.9160895347595215, -0.0690656304359436, -1.2320821285247803, 0.7142771482467651, 0.5559458136558533, 1.57130765914917, -1.3018009662628174, -0.4959847927093506, 0.6598659753799438, 0.2054743617773056, -0.5646739602088928, 0.3858630359172821, -1.2559406757354736, -0.5730627775192261, 1.456424593925476, 0.5814939737319946, -0.10168628394603729, -1.3603769540786743, -0.45422741770744324, -0.10514570027589798, -0.31875595450401306, 1.1891728639602661, 0.9958608746528625, -0.45985206961631775, 0.5502332448959351, -0.661993145942688, 1.2084882259368896, -1.8231700658798218, -0.05486375465989113, -0.5174035429954529, 1.464140772819519, 0.32476362586021423, 0.3983539640903473, -0.26456764340400696, -0.07347072660923004, -1.4534046649932861, 1.4345519542694092, -0.2878304123878479, -1.9069104194641113, 1.5931333303451538, -0.7494010925292969, 0.19063036143779755, -0.5070975422859192, 0.5393524169921875, -0.35230740904808044, -0.11310180276632309, -1.2614983320236206, -3.0392487049102783, 0.20995688438415527, -0.28182318806648254, -0.5730469822883606, -1.0534201860427856, -0.6285901069641113, 0.6975259780883789, 1.0152989625930786, 0.7977500557899475, 0.44382917881011963, 2.328268051147461, 0.5660315752029419, 0.8061953783035278, -1.0312358140945435, 0.22377082705497742, -0.7278066277503967, 1.4793660640716553, -0.7297145128250122, 0.0053972541354596615, -0.4833809435367584, 2.0914132595062256, -1.5594879388809204, -0.029643749818205833, 1.7012901306152344, -0.5493207573890686, -0.8200168013572693, 0.5241697430610657, -0.03896946460008621, 0.7165160179138184, -0.25665825605392456, -0.5782850980758667, -0.5461405515670776, -1.0027610063552856, -2.074146270751953, -0.3054695725440979, -0.013715244829654694, -1.0991309881210327, 1.9996469020843506, 0.2579886317253113, 0.7218676805496216, -1.229089617729187, 0.7219127416610718, -0.08242619037628174, 0.5216400623321533, -2.2090835571289062, -1.4637565612792969, 0.1603403240442276, 0.5052928924560547, -0.20741285383701324, -0.6506336331367493, 0.09014920890331268, -0.5055199861526489, 0.6782437562942505, 0.4061397314071655, -1.982661485671997, -1.7831887006759644, -1.47097647190094, 1.9875811338424683, -0.5232516527175903, -0.38217949867248535, -0.9129547476768494, 1.450222373008728, 1.14142906665802, 1.426229476928711, 1.199650526046753, 0.04216007888317108, -0.4116167724132538, 0.29608049988746643, 0.40300455689430237, 0.3682638108730316, 1.5290133953094482, 1.6874349117279053, 0.17115846276283264, 0.30792927742004395, -0.7189491987228394, -0.6409478783607483, 0.21629880368709564, 0.13688088953495026, 0.25289615988731384, 0.2363153100013733, -1.7236063480377197, 1.436432957649231, -0.047856707125902176, 0.5316645503044128, 1.1139050722122192, 1.5541796684265137, -0.6509407162666321, 1.4964821338653564, 1.718735933303833, 0.2984592616558075, 2.136512041091919, -1.4484385251998901, 1.1392288208007812, -1.2371318340301514, 0.5980918407440186, 0.29482507705688477, 0.023107359185814857, -0.5417208075523376, 0.5181289911270142, 2.0983684062957764, 0.19580860435962677, 0.9840809106826782, -0.9976217746734619, -0.7216179966926575, 1.2144731283187866, -0.9789088368415833, -0.34401801228523254, 0.39970311522483826, 1.4191282987594604, -1.393686056137085, 1.099905014038086, -0.43172043561935425, -0.26061782240867615, 0.9707670211791992, 1.0412472486495972, 0.7179937958717346, -0.5963336825370789, -2.232924222946167, -0.10104406625032425, 0.03788783401250839, 1.5539395809173584, 0.25788137316703796, -2.1704163551330566, -0.24806946516036987, -0.3413132131099701, 0.7189962863922119, -1.2526661157608032, -0.43434837460517883, -1.0410383939743042, -2.7729814052581787, 0.4459758400917053, -0.19173815846443176, -0.013915573246777058, -0.5220350027084351, 0.7424891591072083, -0.4588750898838043, 0.14951011538505554, -0.5373148918151855, 1.727304220199585, -0.7677649855613708, 0.638276219367981, 1.756679892539978, 1.6866289377212524, -0.04806485027074814, -0.7476825714111328, -1.0878034830093384, 1.0449103116989136, 0.359056293964386, -1.8656564950942993, 1.7851641178131104, 0.5418456792831421, 0.6195938587188721, -0.2433783859014511, -1.155592679977417, 2.3417866230010986, 0.7586228847503662, 1.3808950185775757, -0.5702928304672241, 0.857381284236908, 0.1744907796382904, 1.530716061592102, -0.8608141541481018, 1.1972707509994507, -1.2551323175430298, 1.4825451374053955, -0.9930228590965271, -1.0906445980072021, -0.20182810723781586, -0.6909927725791931, -0.19541104137897491], [-0.5876351594924927, 0.6415154337882996, -0.7114693522453308, 0.7079474925994873, -0.25171324610710144, -0.24499940872192383, 0.01681993342936039, 0.600590169429779, -0.03573291748762131, 0.11752668023109436, -0.43988147377967834, -1.2482041120529175, 0.2544059157371521, 1.0138193368911743, -1.2373380661010742, -1.4709978103637695, 1.807054042816162, -0.504722535610199, -0.3130195736885071, 1.4161745309829712, 0.24279405176639557, 0.17890936136245728, 0.14945892989635468, -0.33380988240242004, -2.103254556655884, 0.29330936074256897, -0.5946698784828186, 0.7428305745124817, 0.1776224821805954, -0.27493712306022644, 0.4747970998287201, -0.7528887987136841, -0.26461029052734375, 0.006527340970933437, 1.028225064277649, 1.7872250080108643, 0.49349847435951233, 0.4553172290325165, -1.0625030994415283, -0.11242803931236267, -0.39348629117012024, -0.0710800290107727, -0.7512506246566772, -0.10078570246696472, -0.2012261301279068, -0.23824386298656464, -0.3636380136013031, -0.4706798493862152, 0.6151219010353088, 0.920117974281311, 0.9586373567581177, -0.04972374066710472, 1.2395941019058228, 0.16456188261508942, -0.6529252529144287, -1.503487467765808, 0.8707715272903442, 1.9172694683074951, 0.21864905953407288, 0.2351909726858139, 1.2625505924224854, 0.2835649251937866, -0.6599376797676086, 0.18834076821804047, 0.8915239572525024, 0.457340806722641, 0.3662433326244354, 1.3618005514144897, 0.2490168958902359, 0.492714524269104, -2.6854183673858643, 0.8674028515815735, -1.4287347793579102, 0.7206729650497437, -0.5373209118843079, -0.7663252353668213, -1.1217098236083984, -0.9959415793418884, 1.0439010858535767, 0.6670020818710327, 0.03336133062839508, -1.4502971172332764, 1.1349889039993286, -0.7745704054832458, -1.0489481687545776, 1.673404335975647, -0.92840576171875, -1.0794429779052734, -1.24616539478302, -0.819722592830658, 0.9216998219490051, 0.21188700199127197, -1.929630160331726, -0.14414623379707336, -0.20620977878570557, -0.3111437261104584, -1.0245518684387207, -0.5384598970413208, 0.36953410506248474, -1.5574132204055786, 0.1329619139432907, -1.654132604598999, -0.4877309799194336, -0.25184354186058044, -0.042703934013843536, 1.1793614625930786, -0.05964824557304382, 0.29096946120262146, 0.799079418182373, 0.8694382905960083, -0.34969034790992737, 1.115919828414917, 0.8252565264701843, 2.0812809467315674, -0.5006300806999207, 0.3756162226200104, -0.05964432284235954, -0.06110013648867607, 0.831233561038971, 1.0737937688827515, 0.4097261130809784, 0.8429239988327026, -1.1353099346160889, 0.6780627369880676, -0.5817550420761108, 2.293966293334961, 0.443548321723938, -2.3108971118927, 1.5293138027191162, -0.9565525650978088, -1.7452161312103271, -1.2887639999389648, -0.8677901029586792, 1.2007113695144653, -0.6798138618469238, -0.00255358568392694, -0.9077058434486389, 0.5213422179222107, 0.994299590587616, 0.8772972822189331, -0.32193922996520996, 1.8209874629974365, -1.2407044172286987, -0.8387973308563232, 0.489103227853775, -1.3495348691940308, -0.9490470290184021, 0.6261260509490967, -1.5652116537094116, 0.5471009016036987, -0.1613328456878662, -0.4048435091972351, 2.010270833969116, -1.1593756675720215, -0.8085817098617554, 0.397625207901001, -0.7976264357566833, -1.2237303256988525, -0.7117164731025696, 1.595153570175171, 0.2162548303604126, -0.992475152015686, -0.397289514541626, 0.09118248522281647, -1.1258618831634521, 0.5964165329933167, 0.9196416139602661, 0.7385110259056091, 0.8240943551063538, -0.9596824645996094, -0.12867183983325958, -1.627742052078247, -1.1653807163238525, -0.4299492835998535, 0.444179505109787, 1.148755669593811, 0.7417668700218201, -0.5735749006271362, 0.2528945803642273, 1.0912981033325195, 1.7541409730911255, -1.192587971687317, 0.04677728936076164, 0.32758376002311707, 1.620029091835022, 1.1315263509750366, 0.06127583980560303, -0.6478390097618103, -0.9854410290718079, -1.1848522424697876, 1.8015475273132324, 0.506580650806427, 0.9306941032409668, 0.5943613648414612, 0.24859920144081116, -1.632228136062622, -0.07599128782749176, 1.7256550788879395, 1.0445244312286377, -0.21829548478126526, 1.1036772727966309, 1.4002877473831177, 0.7505363821983337, -0.6175432205200195, -0.2469925731420517, 0.3530159890651703, -0.7338972091674805, 0.2005496472120285, 1.0597667694091797, 0.1965501755475998, 0.960695207118988, -0.3177310824394226, 0.15621539950370789, 0.3354796767234802, 0.42832350730895996, 0.02795330435037613, -0.4393656253814697, -0.6407536268234253, 0.44046255946159363, -1.799047827720642, -1.4647458791732788, 0.260138601064682, -1.574337124824524, 1.6943994760513306, -0.7544918656349182, -0.8460088968276978, 0.4600755572319031, -0.005679646041244268, -0.27821338176727295, 0.12069369852542877, -1.5417039394378662, 1.5376170873641968, 0.20408901572227478, 0.3813481628894806, -1.317121148109436, -0.34216776490211487, -0.29699981212615967, -0.9149155020713806, 0.6494598388671875, 0.04790670797228813, 0.4128775894641876, 0.6617604494094849, -0.1892060935497284, -0.19372247159481049, 0.5644906163215637, -1.2234385013580322, 0.3139525055885315, -1.98153555393219, 1.157785177230835, -1.2709213495254517, 0.7036364674568176, 1.0767934322357178, -0.8640375733375549, 1.3517802953720093, -0.15938548743724823, -0.5069413781166077, -0.4049040377140045, -1.1600918769836426, 1.1274993419647217, -0.6557201147079468, -0.4874196946620941, -0.3973366916179657, -0.3969106078147888, -0.4690402150154114, -0.004475429188460112, -1.604412317276001, 0.4217700958251953, 0.05971484258770943, 0.5310849547386169, 0.7364974617958069, 0.03725074604153633, -0.4299010634422302, -0.6420665979385376, 1.114275574684143, 1.2727205753326416, 0.37695804238319397, 0.17836102843284607, -0.8978789448738098, 0.2563517689704895, 0.7286578416824341, -0.24650391936302185, 1.1480680704116821, 0.5117819905281067, 0.8224596977233887, -0.6143943071365356, 0.5733388662338257, -0.7700429558753967, 0.4299575090408325, 1.0895397663116455, 0.5393190383911133, -1.0017396211624146, 0.711554229259491, -0.7288790345191956, -1.2017470598220825, 1.3482614755630493, 0.13370326161384583, 0.24547754228115082, 0.3226231038570404, 0.490350604057312, 0.7463157176971436], [0.8578073978424072, -1.2171730995178223, -1.8731695413589478, -0.21097971498966217, -1.3891315460205078, -1.154362678527832, -0.8931668400764465, 1.2074310779571533, 0.5863651037216187, -0.2116580605506897, 1.670640468597412, -0.4044210910797119, 0.36270466446876526, -0.003102141199633479, 0.7299927473068237, -1.3740074634552002, 0.08595790714025497, -0.7188297510147095, 0.5015472173690796, 1.1360009908676147, -0.47309160232543945, -0.23650653660297394, -0.2961232364177704, -0.7231380343437195, 2.599210262298584, 0.18182560801506042, 1.6090936660766602, 1.005907416343689, -0.3254726529121399, -1.327704906463623, -0.872128963470459, 0.1617317497730255, -1.33125901222229, -0.9737637042999268, 0.14551295340061188, 0.5032626390457153, 1.0552383661270142, -0.2341058850288391, 1.0841238498687744, 0.7514858245849609, 1.499300241470337, 1.0441712141036987, 0.35920730233192444, 1.834183931350708, -0.1314401477575302, 0.10254096984863281, 0.12898452579975128, -1.698204755783081, -1.414744257926941, -1.1127545833587646, 0.46650180220603943, 1.1264113187789917, 0.22478856146335602, -0.7510814070701599, 0.45242032408714294, -1.2027021646499634, 1.77592134475708, 0.14147382974624634, 0.8933597207069397, -0.8609529137611389, 0.9104061126708984, -0.54442298412323, -0.3256298899650574, -1.1685553789138794, -0.6176075339317322, -0.4800824522972107, -0.8853027820587158, -0.18825244903564453, -0.6622327566146851, 2.0031540393829346, 1.9116445779800415, 0.40810322761535645, -1.5431718826293945, 0.4475349187850952, -0.8226860761642456, -0.018865808844566345, 0.9801910519599915, 0.853559672832489, 1.7643085718154907, -0.18669690191745758, 0.32721295952796936, -0.4877147972583771, 0.17085811495780945, 0.19683556258678436, -1.5925145149230957, -0.6543529629707336, -1.0606343746185303, -1.6605993509292603, 1.7056719064712524, 1.7350351810455322, 1.1925660371780396, 0.2586382329463959, -0.9613412618637085, 0.04157127067446709, 0.07361697405576706, -0.8470443487167358, 0.6505587697029114, -0.6205040812492371, 0.4512757956981659, -0.32303160429000854, 1.0089105367660522, 0.8185319304466248, -1.4940263032913208, -1.3549312353134155, 2.2255847454071045, 1.268983244895935, 0.9154781699180603, -0.11578250676393509, 0.26552698016166687, -2.1518537998199463, 0.20520006120204926, 0.16284343600273132, -0.622865617275238, -1.466721773147583, -1.7369377613067627, 0.6061325669288635, -0.07683774828910828, -1.5735307931900024, -1.3828507661819458, 1.5985599756240845, -0.04237369820475578, 0.9692937731742859, -0.36700311303138733, 1.2699978351593018, 0.9946387410163879, -0.9293003678321838, -0.7737633585929871, 0.902782678604126, -0.9015698432922363, 0.6957975029945374, 0.576228141784668, 0.2463797777891159, -1.5198277235031128, 0.09144628793001175, -1.6146165132522583, 0.9884638786315918, -0.9204497337341309, 0.54671311378479, 0.357476145029068, 0.9650547504425049, -1.4394354820251465, 0.09561823308467865, -0.7915668487548828, 0.04016377776861191, 0.7881020903587341, 0.6287062764167786, -0.8660863041877747, 1.2788711786270142, 1.2541847229003906, 0.9773537516593933, -0.6695199608802795, -1.2523974180221558, 0.26061317324638367, 0.34788379073143005, -1.242904782295227, 0.27594637870788574, -0.763714075088501, -0.20524947345256805, 1.3357503414154053, 1.3245495557785034, 1.6115379333496094, -1.814028263092041, -2.568877935409546, 0.9976964592933655, -0.6605621576309204, 0.30307552218437195, -1.6044507026672363, -0.3673883080482483, -1.056627631187439, 0.9929679036140442, -0.08354301750659943, -0.08953184634447098, -0.9517027139663696, -0.2063884139060974, 1.192438006401062, 0.7065614461898804, -0.08817939460277557, 0.17650122940540314, -0.03347272798418999, 1.042828917503357, 1.2791979312896729, 0.8652735352516174, 0.11026065051555634, -1.6575977802276611, 0.05826583504676819, -1.3118624687194824, 0.6556150913238525, -0.13082922995090485, -1.4831799268722534, -0.30131202936172485, -0.04321752488613129, -1.577078938484192, -0.3030371367931366, -1.4033647775650024, -0.6389860510826111, -0.16997092962265015, -0.44360587000846863, 0.014206547290086746, 0.6547167301177979, 0.5334247946739197, -0.24766431748867035, -1.3644694089889526, -0.3398894667625427, 0.5027442574501038, 0.8933199644088745, 1.2289456129074097, -3.1932547092437744, -1.1954094171524048, 0.3601563572883606, 0.2801050841808319, 1.8196938037872314, -0.22251194715499878, -0.37861084938049316, -2.053036689758301, 0.32171040773391724, -1.4830734729766846, 1.5243887901306152, -0.380009263753891, -1.7307870388031006, -1.1954978704452515, 0.4579862058162689, -1.333229660987854, 0.9574902057647705, 0.14723846316337585, -2.043902635574341, 0.04786042869091034, 2.773977518081665, -0.7842211723327637, -1.9615719318389893, 0.33790212869644165, -0.7607424259185791, 0.1585974246263504, 0.750609278678894, -0.5903821587562561, -1.1991071701049805, 1.172776460647583, 0.015272850170731544, -0.4629738926887512, 1.764115571975708, -0.6146293878555298, 1.3807718753814697, 0.8927220106124878, 1.3898744583129883, 1.2406034469604492, -0.007850772701203823, -0.8538863062858582, 0.5584222078323364, -0.37644368410110474, 1.0851738452911377, -0.7849816679954529, -0.5380282402038574, -1.2293858528137207, -0.9028158187866211, 0.3999303877353668, -1.1950321197509766, 0.34632059931755066, 2.8770813941955566, 1.5019116401672363, -0.2917953431606293, -1.7404630184173584, -0.8207036256790161, 0.8811870813369751, -0.20244593918323517, 1.3209688663482666, 0.24262356758117676, 0.16027013957500458, 1.0659353733062744, -0.33789753913879395, 0.7094771265983582, 1.1900161504745483, -0.440497487783432, 1.420753836631775, 0.7533680200576782, -1.4405854940414429, -0.9198659658432007, 1.4557685852050781, -1.3750629425048828, 0.972201943397522, -0.44649538397789, -0.6756923794746399, -2.081655979156494, -0.7068442106246948, 0.26377594470977783, 0.7516863346099854, -1.3243844509124756, -1.1776328086853027, 0.6704892516136169, 1.3038939237594604, -1.1049628257751465, -2.223249912261963, -0.31922590732574463, -0.2872459590435028, -1.7236995697021484, -0.8095309734344482, 0.7418214082717896, 0.3114039897918701, 0.13286353647708893, 0.3426673710346222, 0.9138039350509644, -0.9466190338134766], [-0.8357742428779602, 0.8979557156562805, 0.005293232388794422, 0.213245689868927, 1.4285610914230347, -0.4020201563835144, 0.03743061050772667, 0.8497528433799744, 2.202064037322998, 2.292030096054077, 0.32182857394218445, -0.36686816811561584, 0.11495528370141983, 0.8563629984855652, -0.6029453277587891, 0.10213926434516907, -0.44156351685523987, 0.038171201944351196, 0.23255619406700134, -1.1500813961029053, -0.020847303792834282, -0.07411852478981018, -0.15773136913776398, 0.7075709700584412, -1.9309815168380737, -0.2713260352611542, -0.3787015974521637, 1.2408647537231445, 0.2128104269504547, 0.6588148474693298, 0.448397696018219, 0.2825479805469513, -0.7493057250976562, -2.20281720161438, -0.4129444658756256, 1.7734858989715576, 2.3440916538238525, 1.3003531694412231, -0.003901855321601033, -0.016664927825331688, 0.10951706767082214, 0.04185062646865845, 0.7391685843467712, 0.9777101278305054, -0.10953156650066376, -1.3704079389572144, 0.37982290983200073, -0.2218717783689499, -0.6718354225158691, 1.0062495470046997, 1.319190502166748, -0.8142062425613403, 0.7279424071311951, -0.4392620921134949, -0.9636713862419128, 0.7914167046546936, -0.11425318568944931, 0.14303185045719147, 0.22694705426692963, -0.10919925570487976, -0.7222247123718262, 0.29625916481018066, 0.038523681461811066, -0.03005397878587246, -0.31534266471862793, 1.0902823209762573, -0.5847805738449097, -1.4503633975982666, 0.764201283454895, 2.561892032623291, 0.5863909125328064, 2.9063825607299805, 0.4255637526512146, -1.2729934453964233, 0.5480282306671143, 0.08387867361307144, -0.7495270371437073, 0.6425988674163818, 0.49276554584503174, -0.046009764075279236, 0.8084849119186401, -0.43073076009750366, 0.6093651652336121, 2.7184433937072754, -0.7826610207557678, 0.23838172852993011, -0.0022124089300632477, -0.1663016527891159, 0.3006305694580078, -0.29489317536354065, 0.6163874864578247, 0.6377485990524292, 1.8010905981063843, -1.3356646299362183, -1.7662028074264526, -0.0019549818243831396, -0.1920066475868225, 0.5458056330680847, -0.5839410424232483, -2.1481387615203857, 0.8913426399230957, -1.5387709140777588, -1.1401853561401367, -1.1837705373764038, 0.725956380367279, 0.8754270076751709, -1.2836920022964478, -1.6575247049331665, 1.1039177179336548, -0.15397951006889343, 0.4231905937194824, 1.047271490097046, -0.0806940570473671, 1.9030896425247192, -2.2676546573638916, -0.86000657081604, 0.19452938437461853, -1.0768845081329346, 0.7143086194992065, 1.8003976345062256, 0.8280244469642639, 0.2192155122756958, 1.36881685256958, -1.315969467163086, 0.9083857536315918, 1.275547981262207, -0.7138970494270325, 0.10604799538850784, 0.019291045144200325, 1.4661072492599487, -0.33014485239982605, 1.6621853113174438, -1.4260823726654053, 0.03060077130794525, -0.009417533874511719, -0.14803506433963776, 0.3498730957508087, -1.8792402744293213, -0.7298979759216309, -1.3005365133285522, 1.1391111612319946, -1.1335066556930542, 0.8451811671257019, -1.2819689512252808, -1.3513293266296387, 1.0380557775497437, -1.4768131971359253, -0.40106600522994995, -1.3998702764511108, -0.4134541451931, -0.36111658811569214, -0.20562294125556946, 0.8589166402816772, 0.8823836445808411, -0.17552585899829865, -0.4857045114040375, -1.297256350517273, 0.23330345749855042, 1.0353587865829468, -0.5205737948417664, 1.5240478515625, -0.3360652029514313, -0.8798840045928955, -0.08750003576278687, 0.42823266983032227, 0.22011636197566986, 0.18279175460338593, 0.4922766387462616, -0.02327193133533001, -0.6858830451965332, 0.5839986205101013, -1.2141326665878296, 1.7136155366897583, 1.0435250997543335, -1.656807541847229, -0.510383129119873, -0.8865876197814941, -1.0596061944961548, -1.074440836906433, -1.0463244915008545, -2.0913684368133545, -1.5926742553710938, -1.391208529472351, 0.7844946980476379, 1.0015136003494263, -0.2739723026752472, 0.36388468742370605, 0.3907298743724823, 0.30223605036735535, -1.1026350259780884, -0.6508923768997192, 0.2826295793056488, -0.6239218711853027, -0.5383079648017883, -1.0736459493637085, 0.539795458316803, 1.6431608200073242, 0.8212011456489563, -0.6466859579086304, 1.6414246559143066, -0.7747144103050232, -2.2364556789398193, -0.43536022305488586, -0.16672319173812866, 0.12645423412322998, -0.031014708802103996, -0.008547132834792137, 0.17732636630535126, 0.8554493188858032, 0.4779720902442932, 0.3462018072605133, -0.6504897475242615, 0.04491707310080528, -1.2024164199829102, 0.4611245393753052, 0.22350408136844635, -1.4194278717041016, -0.6799284219741821, 1.832139253616333, -0.5463250875473022, 0.5333774089813232, -1.644874930381775, -0.8277654051780701, 0.03557705506682396, 0.15952956676483154, 0.28270551562309265, -1.0163488388061523, -0.4437069594860077, 0.5915494561195374, -0.1939697265625, -0.6012932658195496, -1.414912462234497, -0.3541884124279022, -0.179243266582489, 0.807211697101593, 0.204743430018425, -0.033331409096717834, -0.8214045166969299, -1.5672602653503418, -0.34875544905662537, -0.5693880319595337, -1.251833438873291, -1.0396615266799927, 1.2561010122299194, -0.2660081386566162, -1.8554465770721436, 0.6080357432365417, -1.1416093111038208, 0.4271078109741211, -0.13097447156906128, -0.8239892721176147, 2.1699087619781494, 0.05797898396849632, 0.31311193108558655, 1.834628939628601, -1.0713847875595093, -0.5383028984069824, -1.2298907041549683, -1.9263181686401367, -0.8867958784103394, -0.7721160054206848, -1.9478645324707031, -0.5463534593582153, -1.181255578994751, -0.5397859215736389, 0.30534952878952026, 0.4348424971103668, 1.3703514337539673, -1.8255877494812012, -0.5360110998153687, 0.6567453145980835, 0.2211451381444931, 0.25223809480667114, 1.162551999092102, 1.2292479276657104, -0.111728735268116, 0.9090098738670349, -0.3412908911705017, 1.2673511505126953, 1.031446933746338, 1.329158067703247, 0.6922153830528259, 0.7535838484764099, -1.715671181678772, -0.12489280849695206, 1.5779361724853516, 1.045369029045105, 0.06938249617815018, 0.025354789569973946, -0.45202818512916565, -0.9281946420669556, 1.0836646556854248, 2.118311643600464, -2.1501917839050293, 0.65016108751297, 0.7400491833686829, -0.3601744771003723, 1.1360489130020142, -0.37665483355522156, -0.04784919321537018], [1.1541463136672974, -2.9257097244262695, 1.4988818168640137, 1.14462411403656, -0.23847229778766632, 0.8970275521278381, -0.7922139167785645, -1.0985584259033203, -0.21523195505142212, 1.3970146179199219, 2.187941074371338, -1.3063160181045532, -1.1616853475570679, -0.22239521145820618, -0.38612449169158936, 0.48491743206977844, -0.8995913863182068, -0.27302980422973633, 0.4148523211479187, 0.9761502742767334, 0.4442046284675598, -0.8740352988243103, -1.8476020097732544, 2.028111219406128, -0.5574949383735657, -0.0604109987616539, 0.1868070513010025, -0.9038375616073608, 0.1888265609741211, -2.4746265411376953, 0.38533759117126465, 0.6986947655677795, 0.6636853218078613, 0.9041796326637268, 0.35362741351127625, 0.807761549949646, 0.9462810158729553, 0.8612037301063538, 0.028798768296837807, -0.12339284271001816, -0.8653672337532043, 0.1164192259311676, 1.449429988861084, -0.6010546684265137, -0.8111515045166016, 1.4334232807159424, -1.2980499267578125, 0.024006258696317673, -1.5440330505371094, 0.07077761739492416, 0.9384671449661255, 0.08083844929933548, -0.80009526014328, 0.48776930570602417, -1.0305092334747314, -0.12226796895265579, -0.8947985768318176, -2.0919766426086426, -0.7179254293441772, -0.4205016791820526, -0.33126887679100037, 0.8011577725410461, -0.7153149247169495, -1.033240556716919, -0.16940172016620636, 0.6771048903465271, -1.1251178979873657, 2.2138612270355225, 0.4863395690917969, 0.9149105548858643, -0.5523614883422852, 1.400039553642273, 1.3699029684066772, 0.4187057912349701, -0.12502971291542053, -0.9211607575416565, 0.39532944560050964, 0.4595576226711273, -0.7449528574943542, -1.8100435733795166, -0.09915477782487869, 0.5792695879936218, 1.1845989227294922, -0.18416261672973633, -1.9853278398513794, 0.8525813221931458, -1.721911907196045, -0.6261544823646545, 0.24075567722320557, 1.513819694519043, 0.1192508414387703, 0.6521338820457458, -0.3364000618457794, -2.282552719116211, -0.5597567558288574, -2.1006345748901367, 0.5878943800926208, 0.9944588541984558, 0.631980299949646, -0.4075084328651428, -1.5297579765319824, -0.6287477612495422, -0.9212362170219421, 0.11759686470031738, 1.1063499450683594, -0.11601845920085907, -0.1836589127779007, 1.1687089204788208, 0.3283529579639435, -0.7508589625358582, -0.6936648488044739, -0.7327150702476501, -0.9529889822006226, 0.9549096822738647, -0.5632213950157166, 0.045904725790023804, 0.021990610286593437, -0.16340263187885284, 0.16625896096229553, 0.8670178651809692, 0.8676386475563049, 0.39359262585639954, -0.0716744139790535, -0.10281921923160553, 0.28443488478660583, -0.4925796389579773, -0.22510527074337006, -1.8574254512786865, 0.577293336391449, -0.3574187755584717, -1.0117965936660767, 1.5078212022781372, -0.7735188603401184, -0.18976035714149475, 0.5489717721939087, 0.3869576156139374, -0.0885070338845253, -1.9820815324783325, 1.08900785446167, 0.08227812498807907, 1.5021156072616577, 0.8157461285591125, 0.1462181955575943, 1.550194501876831, 0.3466523587703705, 0.22330373525619507, -1.6070386171340942, 0.552652895450592, -0.8063626885414124, 2.0276355743408203, 0.6701630353927612, -0.9167839884757996, -0.42524027824401855, -1.602081537246704, 0.6206656098365784, 2.5627052783966064, -0.29685264825820923, -0.6205519437789917, -1.7362313270568848, 1.0751440525054932, 0.7132652401924133, -0.5987789630889893, 2.0322344303131104, -0.8611310124397278, 0.6439236402511597, -1.6471601724624634, -2.1526122093200684, -0.37076789140701294, 0.1970708817243576, 0.30042093992233276, -1.834004521369934, -0.014080885797739029, 2.0738213062286377, 0.8115304708480835, 0.91871178150177, -0.9751219153404236, -0.7102779150009155, -0.4226996898651123, 0.44097575545310974, 0.45129743218421936, -0.2603141665458679, 0.3925498425960541, -0.7642130255699158, 0.241061270236969, 0.7241379022598267, -1.5289210081100464, 0.13617925345897675, -0.3238174617290497, 3.2109363079071045, -0.20053113996982574, 0.9041253924369812, -1.4771982431411743, 0.39313334226608276, -0.13189993798732758, 0.4690466523170471, 0.4685831367969513, 2.3708927631378174, -0.8643812537193298, 1.0431305170059204, 1.2690943479537964, 1.0191805362701416, -0.567363440990448, 0.7171670794487, 0.2977841794490814, 0.018568193539977074, 0.3158913254737854, -0.30080586671829224, 0.35765841603279114, 0.22613577544689178, 1.116010308265686, 0.6322174668312073, -0.30052852630615234, 1.376608967781067, 1.3990890979766846, 0.4200676679611206, -0.7312179207801819, -2.7565767765045166, 1.2609108686447144, 1.1637593507766724, -0.14790858328342438, -1.0888084173202515, 1.3124829530715942, -0.07212773710489273, -0.9502993822097778, -0.9051722288131714, 1.2031787633895874, 1.8214424848556519, -0.07963340729475021, 0.8410739898681641, 1.0209141969680786, 0.8526580333709717, 1.915279746055603, -1.0830050706863403, 0.7089092135429382, 0.8970022201538086, 0.11352308094501495, -1.3916141986846924, 0.8967899680137634, 0.5880058407783508, 0.14272142946720123, 0.5972892642021179, 1.3165591955184937, 0.9015762209892273, 0.44903770089149475, -0.7422878742218018, 0.7213042974472046, 1.9222121238708496, 0.6377736926078796, -1.9292010068893433, 0.006593963596969843, -0.3730814456939697, 0.01785944029688835, 1.0363556146621704, -1.1670314073562622, -0.5285151600837708, 1.1063952445983887, 0.7977437973022461, 0.4619678258895874, -0.8358646631240845, -1.4635789394378662, -0.22872285544872284, 1.7033509016036987, 0.9868286848068237, 0.4232328534126282, 0.37130099534988403, -1.3382275104522705, -2.3671019077301025, -0.751006543636322, -1.17887282371521, -1.0319948196411133, 0.3322034180164337, 0.45227670669555664, 0.19461217522621155, -1.8580846786499023, 1.3824732303619385, -0.2715660035610199, -1.3383162021636963, -1.9897048473358154, -1.186303973197937, 0.6379280686378479, -0.2791757583618164, 0.5002180337905884, -1.4185503721237183, -0.125803142786026, -1.4235529899597168, -1.0459530353546143, 0.46483686566352844, -0.06322502344846725, -0.7641658186912537, -0.8680757880210876, 2.492027521133423, -0.20913894474506378, -1.0699599981307983, -0.5999974608421326, 0.8063051104545593, -1.4602841138839722, 1.0290573835372925, -0.21981577575206757, 0.045796606689691544, -0.34116441011428833], [-1.4443992376327515, 0.7348507642745972, -0.8687992095947266, -1.328275203704834, -0.6282072067260742, 0.6451047658920288, -0.4075525999069214, 0.856985330581665, -0.5669137835502625, 0.5225123167037964, -1.2271010875701904, -0.935247540473938, 1.565393328666687, -1.4594078063964844, -0.6514776349067688, 0.6884856820106506, 1.3991857767105103, -0.030210373923182487, 0.20411181449890137, 0.6522197723388672, 0.36031970381736755, 0.21354934573173523, -1.438870906829834, -0.13389453291893005, -1.699962854385376, 1.5140177011489868, -0.12648090720176697, -2.046546459197998, 1.7918225526809692, 0.0011783489026129246, -0.48921260237693787, 1.8336758613586426, 0.8266069889068604, 0.9874787926673889, 0.5842902064323425, -0.18611733615398407, 1.0665016174316406, 0.05912591889500618, -1.0461928844451904, 0.34291693568229675, 0.8881519436836243, 0.8970524072647095, -0.08662604540586472, -2.097080707550049, 1.5341148376464844, -0.49265924096107483, 0.36297422647476196, 0.5516006350517273, 1.087961196899414, 0.37344932556152344, -0.1899464875459671, -0.22901096940040588, 1.9952011108398438, -0.4704100489616394, -1.4897304773330688, -1.1696183681488037, 2.2425284385681152, -2.054417848587036, -0.04309607297182083, 0.42787984013557434, -0.5403568744659424, -0.5074165463447571, -0.47533875703811646, 0.2340008169412613, -0.5616689324378967, 0.09518440067768097, 0.3092208206653595, -1.2672829627990723, 0.7348041534423828, -0.6311442852020264, -0.3336775004863739, 1.2993419170379639, -1.2816264629364014, 0.27642717957496643, 0.43533560633659363, -0.44316068291664124, -0.4600437581539154, 0.791182816028595, -0.024719247594475746, 0.03314991295337677, 0.5423676371574402, 0.8861503005027771, -0.6662758588790894, -1.5277395248413086, 1.6112892627716064, 1.4157159328460693, 0.6609569191932678, -0.020090486854314804, 0.6403835415840149, 0.9946957230567932, 0.7174518704414368, 1.4829694032669067, 1.508105754852295, 0.6982558369636536, 0.6734089851379395, 0.5342226624488831, -0.4070526659488678, -0.8436599373817444, 0.33702075481414795, 2.032245397567749, -1.8426554203033447, 0.7644867300987244, 1.7139867544174194, 1.0641573667526245, -1.6425549983978271, -0.22946274280548096, 0.15101054310798645, 0.951313853263855, 0.31688931584358215, -1.0935919284820557, 0.08824416249990463, -0.0074783978052437305, -2.0297610759735107, -0.8293064832687378, 1.1353518962860107, 0.4441061019897461, 0.3117266893386841, -1.7692785263061523, -0.32840579748153687, 1.6930835247039795, 0.13239189982414246, -1.2964154481887817, -1.5363737344741821, 0.7994996309280396, 1.331223964691162, 0.9921298623085022, 0.49575644731521606, 0.35926687717437744, -0.09908414632081985, -0.4584161043167114, 1.9777615070343018, 0.7141878604888916, -0.30625689029693604, -0.984889805316925, 0.7420687079429626, 0.999923586845398, -1.001141905784607, 1.5293867588043213, -0.8416067361831665, 0.3250868320465088, 0.782512903213501, 1.000854253768921, 0.812431812286377, 1.1998568773269653, 0.4575263261795044, -0.32253947854042053, 0.12248189002275467, 1.5943259000778198, -0.4694283902645111, -2.7461626529693604, 0.39136189222335815, -0.44585978984832764, -1.1861799955368042, 0.9715071320533752, -0.6139106154441833, -0.13153335452079773, -0.5308732986450195, -0.4037017226219177, 0.44342970848083496, -0.881131112575531, -0.05765373259782791, 1.3108725547790527, -1.1705588102340698, -0.6988413333892822, -0.7522584199905396, 0.48096367716789246, -0.19166383147239685, -0.6962621212005615, 0.5361406207084656, -0.06383559107780457, 0.8440739512443542, -0.532001793384552, 1.5588399171829224, -0.34517282247543335, -1.2167651653289795, 0.01127969566732645, -1.6605929136276245, -2.1665079593658447, -0.0745273157954216, 1.5318703651428223, 0.4974477291107178, 0.08146766573190689, 0.22224609553813934, -1.2402981519699097, 0.8346742987632751, 1.5976804494857788, 0.4182599186897278, 0.34629109501838684, 0.5416082739830017, 0.5551897883415222, 1.3713018894195557, 1.1803580522537231, -0.28077617287635803, -0.5649402141571045, 0.6748338341712952, 1.9823960065841675, 0.24260811507701874, 0.1010412946343422, 0.7842569947242737, 0.4239773750305176, 0.5655227899551392, -2.2801637649536133, 0.5924693942070007, 0.5247949361801147, -0.45852166414260864, 1.4179556369781494, 0.4984988868236542, -1.2125203609466553, 0.37733420729637146, 0.6323410272598267, 1.5517065525054932, -0.29455816745758057, 1.2127104997634888, 1.3837318420410156, -0.3018447458744049, 0.4471625089645386, -0.6972680687904358, 1.2072596549987793, 0.09902764111757278, -1.1963104009628296, -0.03464978560805321, -1.9452570676803589, -0.08111289143562317, 1.7599180936813354, 0.1731165498495102, -2.340663433074951, -0.6578238010406494, -0.12067165970802307, -0.14407658576965332, -0.5786023139953613, 1.3035074472427368, -2.0280630588531494, 0.4843530058860779, 0.39824485778808594, 0.09481687098741531, 0.22925429046154022, -0.5577906370162964, 0.295296311378479, -1.8337758779525757, -1.025406837463379, 0.8655773401260376, -0.9039347171783447, 0.6502819657325745, 2.321795701980591, -1.7259711027145386, 0.16823016107082367, 1.7559194564819336, -0.113816037774086, -1.5783673524856567, 1.1140817403793335, -1.2615162134170532, -1.6246874332427979, -1.620532751083374, 0.7095663547515869, -0.9495565295219421, -0.24589018523693085, 0.8584661483764648, 1.5393271446228027, -0.5776574015617371, -1.5858433246612549, 2.1357853412628174, 1.6513372659683228, 0.29443588852882385, 1.7170778512954712, -0.36576583981513977, -0.11705921590328217, 0.007267330307513475, -1.47687828540802, -0.12614023685455322, 0.43739601969718933, -0.20818345248699188, 0.18735305964946747, -0.08265992254018784, 0.2625402510166168, -0.08745452761650085, -0.07192955166101456, -0.6386892199516296, -0.46997925639152527, 0.06088826805353165, -0.658319354057312, 0.5956538319587708, 1.3566728830337524, -1.1566141843795776, 0.8244444131851196, 0.42005646228790283, 0.06558263301849365, 0.41718438267707825, -0.2811603844165802, -0.1910986304283142, -0.30705031752586365, -0.3017841577529907, 0.5306095480918884, -1.6172904968261719, 0.7000001072883606, -1.4548546075820923, 0.6686626076698303, 1.106139063835144, 0.5152621269226074, 1.2670382261276245, -1.045127511024475], [-0.2035129964351654, -0.1474090814590454, 1.2326855659484863, 0.3972984850406647, 0.6020038723945618, 0.5672280192375183, 0.22034072875976562, -0.8025438189506531, -1.9938876628875732, -0.5237676501274109, 1.1349427700042725, 0.04462375119328499, -0.30317404866218567, 2.0559165477752686, 0.5081261992454529, -0.41923823952674866, 0.7400816082954407, 0.3242117166519165, -0.04539203643798828, -0.23776423931121826, -0.7414594888687134, 0.08490121364593506, -0.23518294095993042, 1.1278809309005737, -1.0594696998596191, 0.5835871696472168, -0.709826648235321, -1.3671692609786987, -0.38167932629585266, -0.08043830841779709, -1.300572156906128, 0.314358115196228, -1.8824326992034912, -0.7422400116920471, 0.17869870364665985, -0.6443755030632019, -0.21542520821094513, -0.9126933813095093, -1.6593340635299683, -0.8860204815864563, -0.7864617109298706, 0.2503216862678528, -0.3863097131252289, -0.6883180141448975, -1.1205229759216309, 0.6173208951950073, 0.14559921622276306, -0.4223150312900543, 0.36745405197143555, -0.555151104927063, -1.114277720451355, -1.643568754196167, 0.3004036247730255, 1.9076184034347534, 0.14209811389446259, -0.09073111414909363, 0.22193679213523865, -0.031329818069934845, -2.309784173965454, -0.39606955647468567, 0.4692508280277252, -0.9301906824111938, -0.8929235935211182, 0.3508775234222412, 0.6087413430213928, 0.895024836063385, 0.47744378447532654, 0.5722118616104126, -0.5049289464950562, -0.1249585673213005, 0.8054757714271545, -0.7006890773773193, -1.2915077209472656, 0.8491548895835876, 0.4037668704986572, -0.7724511027336121, 0.10926403105258942, -0.5602037906646729, 1.2108887434005737, 1.145127296447754, 0.20225004851818085, -0.49714434146881104, -0.2771710753440857, 1.16999089717865, -0.15386873483657837, -1.0254656076431274, 2.8031411170959473, 0.2136872261762619, -0.3080321252346039, -0.6093644499778748, -0.9068781733512878, 0.7930580973625183, -0.30873987078666687, -0.6935554146766663, 0.7657890319824219, 0.4665648341178894, 0.38617074489593506, 0.5544124841690063, -0.021449869498610497, 0.9985707402229309, -1.1631940603256226, -0.25848135352134705, -0.4294109046459198, 1.8909683227539062, -1.016050100326538, -1.0344339609146118, 1.4544878005981445, 0.7015219926834106, 0.9585261940956116, -1.9304845333099365, -1.2770191431045532, 0.12903611361980438, 0.11788167804479599, 0.5050246715545654, 0.8696128129959106, -0.41682684421539307, 0.5194949507713318, -0.7900172472000122, 1.0886584520339966, -1.3133677244186401, 2.2869873046875, 1.0374321937561035, 0.6124534010887146, -0.5064388513565063, -0.12359489500522614, 0.24866953492164612, 0.6023804545402527, 1.9968150854110718, -0.735061764717102, -1.5315834283828735, 1.7744053602218628, 1.1716573238372803, -0.9670683145523071, 1.458780288696289, 0.19704705476760864, 2.4991602897644043, -1.7027053833007812, -0.5434818267822266, -0.6198209524154663, 0.7440066337585449, 0.8651745915412903, -0.26936063170433044, 0.6618165969848633, -0.3402777314186096, 0.37734997272491455, -0.9648690223693848, -1.4820526838302612, -0.37136271595954895, -0.5433449745178223, 1.33201265335083, 0.5513523817062378, 0.9395589232444763, 2.5440893173217773, -1.1001266241073608, 2.2260847091674805, -0.30159229040145874, 0.8454511165618896, 0.6046940684318542, 0.018953857943415642, 0.09740953147411346, -0.30018875002861023, 0.21193929016590118, 0.4678136110305786, -0.19702641665935516, 1.7505451440811157, -1.023842453956604, 1.7135976552963257, -0.5468025803565979, -0.8813377022743225, 0.06260747462511063, -1.0108391046524048, 0.4776725769042969, 0.9456536173820496, 0.3465512990951538, 1.2715694904327393, 0.8891885280609131, -1.3384270668029785, -0.8560988306999207, 0.8561912775039673, 1.9537773132324219, -2.067953586578369, -0.16356822848320007, -2.96232271194458, -1.2896760702133179, 1.3086315393447876, 0.7947377562522888, 0.7287425398826599, -0.8482377529144287, 0.956900417804718, 1.1401830911636353, 0.9445145130157471, 1.5080056190490723, 0.4570530951023102, 0.827031135559082, 0.30846789479255676, -0.334900826215744, 0.3340189456939697, -0.5172827839851379, 0.43846777081489563, 0.7168843746185303, -0.9740660190582275, 1.7261000871658325, 1.500087857246399, 1.3193846940994263, -0.8335019946098328, 0.5664255619049072, 0.2775486707687378, -1.16610848903656, 2.4094579219818115, -1.822266936302185, 0.4955078065395355, -0.19088228046894073, -0.826903760433197, -0.6041522026062012, -0.9693114161491394, 0.9951577186584473, -0.4617328643798828, 0.3623732328414917, 0.5002442002296448, 0.9130539894104004, -0.0464574471116066, 0.40441909432411194, -0.28943929076194763, -0.8577819466590881, 0.45113158226013184, -0.5210874676704407, -0.5114118456840515, 0.8681230545043945, 1.858677864074707, -0.1531870812177658, -1.0363353490829468, -0.8091650009155273, -2.2027642726898193, 0.5086663961410522, -0.635917067527771, -0.13202466070652008, -1.2706961631774902, 0.8332889080047607, 0.6252754926681519, 0.8325623869895935, -0.508746325969696, 1.4709901809692383, -0.23570367693901062, -1.182526707649231, 0.13609150052070618, -0.24460139870643616, -0.4937002658843994, -1.0652517080307007, 1.0639013051986694, 1.0625416040420532, -0.23183278739452362, 1.1249088048934937, -1.2941431999206543, -0.5152090191841125, -0.457843154668808, -0.7006367444992065, 0.9161387085914612, 0.10843215882778168, 0.7187737822532654, 0.3635577857494354, 0.4125863015651703, -0.0069573912769556046, -0.3275257647037506, -1.6108477115631104, -0.16486027836799622, -0.7879819869995117, -2.163696527481079, -1.0836340188980103, -0.8757955431938171, -0.8982438445091248, -1.476789116859436, -0.8155571818351746, -0.44655606150627136, -0.5771276950836182, 0.2359197735786438, -0.7535086870193481, -0.787365198135376, 0.47130826115608215, -0.38561323285102844, -0.36081424355506897, 0.2668372690677643, 1.123728632926941, 0.386300653219223, 0.27949172258377075, -0.40837132930755615, 0.9303371906280518, 1.443684697151184, -0.9859232306480408, 0.32914605736732483, 0.9916540384292603, -1.869410753250122, 0.9736744165420532, 0.20432177186012268, -0.5600659251213074, -0.428377628326416, 0.3615204989910126, -0.9097152352333069, 1.1319659948349, 0.1081942468881607, -2.514134407043457], [1.6228668689727783, 0.36510348320007324, 1.183115005493164, 1.5570347309112549, -1.2304819822311401, -0.7537879943847656, -0.6831179261207581, -0.4414719045162201, -0.23931799829006195, -0.11325851082801819, 1.2296006679534912, -1.9256510734558105, 0.7092624306678772, -0.7389706373214722, -1.0076302289962769, -0.6401203870773315, -0.6196950674057007, 0.6978200078010559, -0.12473828345537186, -0.37090274691581726, 0.4949394762516022, 1.2431941032409668, 1.6617909669876099, 0.04137159138917923, -0.5166276097297668, -0.6798139810562134, -0.12152126431465149, -0.3812873363494873, -0.9887436628341675, 0.1567796915769577, -1.4518535137176514, -0.5041856169700623, 1.1787769794464111, 0.1000150814652443, 0.07673759013414383, -0.7627412676811218, 0.35253021121025085, 0.9722726941108704, -0.3458831310272217, 0.30170533061027527, -0.09281713515520096, -1.4372880458831787, -0.8274215459823608, 0.7290551066398621, -0.6883959174156189, -0.8400956392288208, 1.0075167417526245, 0.4314711391925812, 1.2185873985290527, -0.08378937840461731, -0.17645490169525146, -0.48696982860565186, -1.5823304653167725, 1.280321478843689, -1.0741459131240845, -0.9830199480056763, 0.30080798268318176, -0.22839336097240448, 0.5150829553604126, 2.466761827468872, -0.9366289377212524, 0.4774296283721924, -0.16916833817958832, 0.03627863526344299, 1.1358476877212524, -0.9270036816596985, -1.5156729221343994, -1.4486644268035889, 0.13370905816555023, -1.6391512155532837, 1.0665266513824463, -0.34870338439941406, -0.10514398664236069, 4.148937225341797, 0.8391278982162476, -0.17296966910362244, -1.4241504669189453, -0.5397121906280518, 0.5537413954734802, -0.8937927484512329, 1.2462300062179565, 0.3484704792499542, 0.2555013597011566, 1.4172621965408325, 1.2294942140579224, 0.5991445183753967, -0.06179935857653618, 0.009776458144187927, 0.1908460259437561, 0.7718048095703125, -0.16608959436416626, 0.7960214018821716, 0.7885177731513977, -0.4490811824798584, -1.480581283569336, 1.0091432332992554, -0.46878182888031006, 0.007828805595636368, -1.4834705591201782, -1.3818957805633545, 0.1868579089641571, 1.1758289337158203, -1.7971200942993164, -1.3216986656188965, -0.33604225516319275, -0.21836550533771515, -0.016102861613035202, 0.15314722061157227, -0.2638838589191437, 0.7071492075920105, -1.310900092124939, -0.32737159729003906, -0.7796756625175476, -2.108826160430908, 0.8540378212928772, 0.31704968214035034, 0.4938458204269409, -0.7365102767944336, -0.38496795296669006, 1.4494354724884033, -0.5278311371803284, 0.2517527937889099, -0.4143053889274597, -1.7744014263153076, -1.7148629426956177, -1.0652209520339966, -0.19958561658859253, -1.3661596775054932, -0.9075388312339783, 0.35853251814842224, 0.7934033870697021, -2.849006175994873, 0.32450127601623535, 0.18083111941814423, -0.1612640917301178, -0.7219753265380859, 0.5657581686973572, 0.061526887118816376, -0.284951776266098, -1.2974132299423218, -0.10390543192625046, -0.4221244752407074, -0.5208078026771545, 1.2549669742584229, 0.08690524101257324, -0.03352855145931244, 0.300428181886673, -0.5642465949058533, 0.36242857575416565, 0.8925339579582214, 0.9540762901306152, 0.03776981309056282, -0.15106596052646637, -0.11775907129049301, -0.3001287281513214, -1.0636626482009888, 0.2596414387226105, -0.9125705361366272, -0.6808589100837708, -1.3894323110580444, -2.365980386734009, 0.6655818223953247, 0.34193068742752075, -0.534898579120636, -1.7611361742019653, -1.0864884853363037, 0.0019285962916910648, 0.17896604537963867, -0.25534185767173767, -1.0569742918014526, -1.1463569402694702, -0.13115093111991882, -0.5319110155105591, 1.1363327503204346, -0.6081945300102234, 1.8689444065093994, -0.24714824557304382, 0.025626692920923233, -0.06598804146051407, -2.2947475910186768, -0.039778389036655426, -0.4420483410358429, 0.11940362304449081, -0.5413697361946106, 2.5103816986083984, -0.8978707790374756, 0.25256291031837463, -1.7323297262191772, -1.216773509979248, -1.1578706502914429, 0.16356948018074036, 0.4742279052734375, 0.8388211131095886, -0.6400055885314941, 0.7254557013511658, 0.46205225586891174, 1.2026883363723755, 1.0814776420593262, 0.7692736983299255, 0.9957293272018433, 1.3187010288238525, 1.0542348623275757, 1.8439357280731201, -0.3676360249519348, 0.321095734834671, 0.9200472235679626, 0.3742028772830963, -0.6505081057548523, -1.0569887161254883, -0.37287482619285583, 1.0648419857025146, 0.298009991645813, -0.062490008771419525, -0.21901972591876984, -1.0887995958328247, -1.1107386350631714, 1.2781052589416504, 0.3889593482017517, 0.29666247963905334, -0.9511027336120605, -0.3091720938682556, 0.9512054920196533, 1.001970887184143, 1.3697866201400757, -0.6854792833328247, -0.43993040919303894, -0.9431427121162415, 1.1388776302337646, 0.02647579461336136, 0.02011100947856903, -2.281296968460083, 0.20465931296348572, 0.9680085182189941, 0.42825397849082947, -0.16121870279312134, 1.340498447418213, -0.37253662943840027, 0.7261293530464172, 0.4069228768348694, -0.48352864384651184, 2.7283313274383545, 0.39448538422584534, 1.203493595123291, -0.9394192099571228, 0.2588540315628052, -0.5394288301467896, 0.6721470952033997, 0.5449422597885132, 0.43071940541267395, -1.3580106496810913, -0.21107982099056244, -0.030604401603341103, 1.2419875860214233, 1.5429891347885132, -1.7689933776855469, -0.5804888010025024, -0.5347276329994202, 0.7030237913131714, 0.532142162322998, 1.112732172012329, -1.1112782955169678, -0.5805178880691528, -0.42164453864097595, -0.427053302526474, 0.8408206701278687, 0.5366763472557068, -0.6197689175605774, -0.8085575103759766, -1.0942935943603516, -0.9390553832054138, -1.0419819355010986, -0.45321282744407654, -1.3943151235580444, -1.2037932872772217, 0.20738954842090607, 0.45392295718193054, 0.5914586186408997, 0.5572698712348938, 0.19902104139328003, 0.39528223872184753, -1.1411687135696411, -0.37770822644233704, 0.6068241596221924, 0.6450710296630859, 1.130897045135498, 0.7389209866523743, 1.4420784711837769, -0.27261775732040405, 0.559225857257843, -0.07114085555076599, -0.5231595039367676, 0.275754451751709, -1.9903662204742432, -0.4956012964248657, 0.5280435085296631, -1.2959544658660889, -0.02330874092876911, 0.2932838499546051, -1.2717726230621338, -0.6337370872497559], [0.10412243753671646, 0.46065714955329895, 1.4226651191711426, -0.8040323257446289, 0.25019949674606323, 0.8789793848991394, 1.003942608833313, -1.8821253776550293, -0.018357504159212112, 1.2412793636322021, 1.8201440572738647, -0.5477180480957031, -0.5848771333694458, -1.352073311805725, 0.7999454736709595, -1.4729396104812622, 0.3147609829902649, 0.382535457611084, -0.09521204978227615, 0.4214361310005188, -0.42017826437950134, 0.7150838971138, 0.41812026500701904, -0.032348353415727615, -2.090137004852295, 0.5536691546440125, 1.32460618019104, 0.7791048884391785, 0.228793665766716, 1.8442386388778687, 1.0960383415222168, 1.0165681838989258, 1.327568769454956, 1.532029628753662, -1.1338881254196167, -0.1496727615594864, -0.3880159556865692, 0.8052141070365906, 0.10571961849927902, 0.746953547000885, -0.761293351650238, 0.10405422747135162, 1.1224849224090576, -0.67127925157547, -1.3978158235549927, 0.01822260022163391, -0.9050360321998596, -0.22476616501808167, 0.10741207748651505, 0.4941403269767761, 0.5544982552528381, -0.2194356769323349, 0.2893601953983307, -0.9803312420845032, -1.31784188747406, 0.7879771590232849, 0.058539748191833496, -0.8381514549255371, -0.7367268800735474, 1.3192545175552368, 0.6990451216697693, -0.4504227042198181, 0.23405160009860992, 0.4787241518497467, -0.06651309132575989, -0.7439031004905701, 0.9664621949195862, 1.7148940563201904, 0.7062217593193054, -0.5484821796417236, 0.383783757686615, -1.3213508129119873, -1.186282753944397, -0.9566736221313477, 0.2013978362083435, 0.4760541617870331, -0.19341515004634857, -1.7532973289489746, 0.3735269010066986, -0.031114840880036354, 0.37489694356918335, 1.2024683952331543, -0.6879385113716125, 3.6889395713806152, 0.02649722807109356, -0.5507847666740417, 0.2717781960964203, -0.3832419812679291, 1.3752132654190063, 0.7463151216506958, 0.7795491814613342, -2.785983085632324, -0.04366494342684746, -0.011634467169642448, 0.2862280607223511, -1.2364726066589355, 0.25629693269729614, 0.9234733581542969, 0.9400925040245056, -0.7511353492736816, 0.6063377261161804, 0.7745999693870544, 0.330048143863678, -0.6518070697784424, 0.902391254901886, 0.39824584126472473, -0.06441443413496017, -0.02908744290471077, 0.9275693297386169, 0.46836522221565247, 0.19548869132995605, -0.8718144297599792, -0.7867839336395264, 0.36935970187187195, 0.9387180805206299, 0.04674743488430977, -1.1411646604537964, -0.9612643122673035, 0.13537386059761047, -0.12167072296142578, 1.4184635877609253, -0.1608220487833023, -1.2053248882293701, -0.20273181796073914, 0.04713889956474304, -0.7752924561500549, 0.5123873949050903, 1.2194069623947144, -1.1317291259765625, -1.1591334342956543, -1.3035969734191895, -0.641743540763855, 0.42019569873809814, -0.7881685495376587, -1.5836739540100098, 1.9253121614456177, 1.3161929845809937, -2.237078905105591, 0.8659430146217346, -0.19754146039485931, -1.5522990226745605, 0.0784740000963211, 0.07821591943502426, 0.9011908173561096, -0.06123165041208267, -1.17318594455719, -0.6302398443222046, -1.6590439081192017, 1.324177861213684, 0.37411296367645264, 1.1054916381835938, -1.7719858884811401, 0.3740735352039337, -0.09561572223901749, 0.20802700519561768, 0.0031097892206162214, -0.42827537655830383, -0.23309925198554993, -0.9284864664077759, -0.5776705741882324, 2.430119752883911, -1.1389371156692505, -0.04065030440688133, 0.8831450343132019, 1.3819141387939453, 3.0463907718658447, 0.3371962904930115, -0.21042805910110474, 0.018848009407520294, 0.7223217487335205, -0.6187027096748352, 0.9985430240631104, -1.2995846271514893, 0.2194836139678955, -0.3481905162334442, 0.2800025939941406, 1.4179649353027344, -0.935785174369812, -0.20576077699661255, 2.5846872329711914, -0.4838728904724121, -0.2726501226425171, 0.2166277915239334, 1.1252875328063965, -2.4140853881835938, 0.2297755479812622, 1.526339054107666, -0.22840535640716553, -1.509723424911499, 0.6574504971504211, -0.37939590215682983, 0.6382166743278503, 1.1484935283660889, 0.5690076351165771, -0.9659633040428162, -0.08773887157440186, -2.954820394515991, 2.0492138862609863, -1.353717565536499, -1.1339112520217896, 0.7583118081092834, -0.6407790184020996, -0.6821482181549072, 0.7807472944259644, -0.7304867506027222, 0.5532697439193726, 0.5795982480049133, 1.080885410308838, -0.9307740926742554, 0.8860083222389221, -0.527734637260437, 0.037849824875593185, -0.300823837518692, -0.5981537699699402, 1.329514980316162, 0.7194246053695679, -1.25074303150177, 0.3785794675350189, -0.2871077060699463, -0.8189509510993958, 2.3386666774749756, 0.616041898727417, 1.4539750814437866, 0.5104639530181885, -0.03650075942277908, 2.2106730937957764, 0.8138225078582764, -0.19404588639736176, -1.4368202686309814, -0.3982306718826294, 1.913824200630188, -0.46406200528144836, 0.027743885293602943, -0.6508585214614868, -0.40115103125572205, 0.41667041182518005, -0.8411075472831726, -1.9455755949020386, 0.7419231534004211, 0.7038975954055786, 0.6354994177818298, 0.9006589651107788, 0.02306833676993847, 0.8849734663963318, 1.3780412673950195, -0.5017120838165283, -0.38379403948783875, 0.08145324140787125, -0.369351327419281, -1.5911544561386108, 0.17812855541706085, 0.6660627722740173, 0.2678009867668152, 0.6488857865333557, -0.14886276423931122, 1.4285303354263306, -0.32815390825271606, 0.6989409923553467, 0.18210801482200623, -0.6609501242637634, -1.4290473461151123, 0.7121880054473877, 0.06952756643295288, 0.45929163694381714, -0.6987544298171997, -0.857172966003418, -0.45874670147895813, 0.8141787648200989, 0.2841881513595581, -0.6704760789871216, 1.4485927820205688, 1.2708760499954224, 1.9198904037475586, 0.32391709089279175, -0.7084074020385742, 0.7577792406082153, -1.6492938995361328, -0.13397829234600067, 0.10284709930419922, 0.4123258590698242, -2.0832393169403076, -1.0994678735733032, -0.815517246723175, 1.6455543041229248, -0.14225751161575317, -0.07155738025903702, -0.4042397141456604, 0.09084154665470123, -0.34795069694519043, -0.5327480435371399, 0.03707779571413994, 0.4664509892463684, 1.171640396118164, 0.35481002926826477, 0.6177540421485901, -0.721592128276825, 0.7557944655418396, -2.4037387371063232, 0.4793955087661743, -0.4892340302467346], [0.47817909717559814, -0.27954864501953125, 0.7049176692962646, -0.7454186677932739, -0.389693021774292, 1.2567763328552246, 2.807683229446411, 1.043279767036438, -0.7779366374015808, -0.868364155292511, -0.9925433397293091, -1.643819808959961, 0.3439464867115021, 0.45640960335731506, -0.4079185426235199, 0.6164107918739319, -0.06219182163476944, -0.7414296865463257, 1.5792465209960938, -0.0455908440053463, -0.1932036131620407, -0.23034454882144928, 0.9119963049888611, -0.4245363175868988, 1.403135895729065, -0.17225003242492676, -0.9087638854980469, 0.2335963100194931, 0.10302092880010605, -3.19866681098938, 0.8397993445396423, 0.1685754358768463, -1.2369307279586792, -0.49919530749320984, 0.36632996797561646, 1.3364733457565308, -0.9347963333129883, 0.4839537441730499, -1.3320949077606201, -0.9054107069969177, 0.33869147300720215, 0.6734121441841125, -0.4322677552700043, -0.7092373371124268, 2.012484312057495, -1.535883903503418, 1.3570464849472046, -0.4981069564819336, 0.49257174134254456, -0.13025692105293274, 0.26641207933425903, 0.22716720402240753, -2.003804922103882, -1.1458054780960083, 0.42751482129096985, -1.2783008813858032, -0.7578418254852295, 0.5751854777336121, 0.8909136056900024, -1.2134093046188354, -0.16359837353229523, 0.350134015083313, 0.9352942705154419, -0.34897875785827637, 0.7286123037338257, 0.5128228664398193, 0.016110679134726524, -0.30012246966362, -0.6323699355125427, -0.9881615042686462, -0.6328997015953064, 0.8861881494522095, -0.4481278359889984, 0.31092944741249084, 0.23486259579658508, -2.5029354095458984, 0.3840234875679016, -0.6756972670555115, 0.8872227072715759, 0.06513863801956177, 1.344188928604126, 0.6387940645217896, 0.46636539697647095, 0.536028265953064, 0.7520943880081177, -0.311816930770874, 0.11459807306528091, 0.30278968811035156, 1.106608271598816, 1.7642008066177368, 0.23221014440059662, 0.9583032727241516, -0.9856962561607361, -0.8362706899642944, 0.6388677954673767, 1.2584526538848877, 0.755046546459198, 2.2652878761291504, 0.27288588881492615, -0.4570843279361725, -0.4711426794528961, 0.6157445907592773, 1.658358097076416, -0.13570190966129303, -1.1222259998321533, 1.7478363513946533, 0.45029401779174805, 0.3279372751712799, 0.01809127815067768, 1.114111065864563, -0.7780181169509888, -1.1241934299468994, -0.8152683973312378, -0.26645636558532715, -0.5660280585289001, -0.35329529643058777, -1.3497529029846191, -0.15909302234649658, 0.19688157737255096, 1.6776725053787231, 0.6360887885093689, 0.30886074900627136, 1.2809916734695435, 1.3845903873443604, 0.23091453313827515, 1.534759283065796, -0.7925151586532593, 0.2106146663427353, 0.5778359174728394, -0.5162509679794312, -0.13300952315330505, -0.1312573403120041, -0.39957547187805176, -1.034730076789856, 1.1245036125183105, -1.3664379119873047, 0.8307293057441711, 1.1755791902542114, 0.7903883457183838, -0.4217526316642761, 0.04134153574705124, 1.5754423141479492, -0.7755210995674133, 0.3303075134754181, -1.6330816745758057, 0.8997183442115784, -1.3340119123458862, 0.25038251280784607, 0.9549345374107361, 0.08425110578536987, -0.24405252933502197, -1.1619775295257568, -1.0762439966201782, 0.7664676904678345, 0.1501474678516388, 0.5962609648704529, -2.6342861652374268, -0.8702545166015625, 0.6119475364685059, 0.7869276404380798, -0.5183472037315369, -0.567154586315155, -0.4249080717563629, -1.8557149171829224, 0.1886235624551773, -0.27132275700569153, 1.8767451047897339, -0.11660952121019363, 0.9341099858283997, 0.23050391674041748, 0.22061343491077423, 0.9251548051834106, 0.3175041377544403, 1.930379867553711, -0.5560986995697021, 1.0147119760513306, 0.3149271607398987, 0.1768064945936203, 0.3578771948814392, -0.4721807539463043, -0.6977719068527222, 1.6608926057815552, 0.9936541318893433, 0.535934567451477, 1.2076947689056396, -1.0821304321289062, -0.4490620791912079, 1.2798219919204712, -1.5403884649276733, 0.6248375773429871, -1.1923874616622925, 0.5823137760162354, -0.4834858179092407, -0.21452856063842773, -1.2542201280593872, 0.5434784293174744, -0.4293423593044281, -0.4007268249988556, -2.4232640266418457, 1.1402703523635864, -1.081344485282898, 1.0490590333938599, -0.5630666017532349, 0.01355199795216322, 1.5174578428268433, 1.07322359085083, 1.3320400714874268, 0.8689092993736267, 0.829308032989502, -1.0745536088943481, -1.0357533693313599, 0.0069138179533183575, 0.1924436241388321, 0.05615251138806343, 0.4590413272380829, 0.7493321299552917, -0.75893634557724, -0.19256485998630524, -0.017706863582134247, 0.6051850914955139, 0.5730122327804565, 0.8344459533691406, 0.35058289766311646, 0.5362511277198792, 0.2804781198501587, 0.06942403316497803, 0.3630170226097107, -1.2267653942108154, -0.8511516451835632, -1.1411747932434082, 0.08951520174741745, -0.05978190526366234, 0.8361856937408447, -1.556227207183838, 1.533937931060791, -0.3378602862358093, -0.02124084159731865, -0.6207531690597534, -0.6933999061584473, 0.5668778419494629, 1.811060905456543, 0.4624311625957489, 0.29184481501579285, -0.7122465372085571, 1.4681141376495361, 1.9442431926727295, -0.4549369215965271, -0.3385278880596161, -0.37956562638282776, -0.7576800584793091, -0.3316342234611511, 0.1866881102323532, -0.6991230845451355, 2.0746707916259766, -0.9281438589096069, 1.7489336729049683, -0.1576436460018158, -0.12593460083007812, 0.04892358556389809, -0.03618312627077103, -0.9371405839920044, 0.012144103646278381, -0.29160064458847046, 1.7153213024139404, 0.8420348167419434, 0.24583743512630463, 1.6360360383987427, 2.555936098098755, 0.4995703399181366, -0.950847327709198, 0.37092116475105286, -1.496846079826355, -0.7218372821807861, -1.1769483089447021, -0.4863380789756775, -0.8933455944061279, 0.7472367286682129, -1.3300379514694214, -0.8221609592437744, 0.3915093243122101, -0.8548614382743835, 1.223222255706787, 1.7609556913375854, -0.743423342704773, 0.6691491007804871, 0.5921791791915894, 0.4008806049823761, -1.5529794692993164, 0.625209629535675, -0.18674297630786896, 1.0687931776046753, -1.255703330039978, -0.48919302225112915, -0.3854825496673584, 1.2365878820419312, -0.3972673714160919, 1.0389750003814697, 0.12926535308361053, 0.7407119870185852, -0.42812031507492065], [-0.47857680916786194, -0.5365291237831116, -0.06515303254127502, -0.03235969319939613, 0.06476549059152603, -0.03927582874894142, 0.07156995683908463, -1.0715128183364868, -0.1399509608745575, -1.1278923749923706, -0.6479844450950623, 0.02604048326611519, -0.5426557660102844, 1.494749903678894, 0.25469323992729187, -0.9280927777290344, 0.026443932205438614, 0.7467889189720154, 0.544848620891571, 1.0519109964370728, 1.7951133251190186, 1.0457135438919067, -1.9713541269302368, 0.23555146157741547, -0.9889171123504639, -0.9495344758033752, -0.7977348566055298, 0.05443084239959717, 0.15568029880523682, 0.8273040652275085, 1.5783593654632568, 1.9138106107711792, -1.2969865798950195, -1.5841259956359863, -1.6014653444290161, 0.3133362829685211, 0.3590473532676697, -1.2061927318572998, -0.21505069732666016, -0.34669071435928345, -0.33384066820144653, 0.31024354696273804, -1.216356873512268, -0.1756991595029831, 2.202200174331665, -0.6064414381980896, -0.20649248361587524, 0.016716955229640007, 0.450162410736084, 0.84646075963974, 0.4947092533111572, 1.1096389293670654, 1.0013253688812256, -0.8951093554496765, -0.23274776339530945, -0.3420877456665039, 0.6091465950012207, 1.5920910835266113, 0.4365280866622925, -0.020969903096556664, 0.09031572192907333, 0.36410194635391235, 0.03225448355078697, -1.2643719911575317, -1.5768826007843018, -0.10151150077581406, -0.8150292634963989, -0.719315767288208, 0.04192616045475006, -0.9431331157684326, -1.3237183094024658, 0.7587879300117493, 1.1768385171890259, -0.3601345121860504, 1.341677188873291, 0.1333584189414978, -0.32630929350852966, 0.16099736094474792, 1.6906427145004272, 0.36731192469596863, -1.550499439239502, 0.52458256483078, -0.23585854470729828, -0.07504192739725113, 1.6821702718734741, 1.1048190593719482, 0.6555839776992798, -0.39585021138191223, -0.5008195042610168, 1.0513843297958374, -0.22583357989788055, 0.8586751222610474, 1.3642505407333374, 1.2026519775390625, -0.5916757583618164, -0.16825152933597565, -0.36428219079971313, 1.0632438659667969, 0.24543894827365875, 1.0842983722686768, 0.5778293609619141, -0.09831225126981735, 0.8381747007369995, -0.7396198511123657, -1.4713561534881592, -0.39048054814338684, -1.4682376384735107, 1.6799166202545166, -1.3032848834991455, 0.12093417346477509, -0.34864920377731323, -0.22585779428482056, -0.7266440987586975, 1.3437414169311523, 0.7239830493927002, -0.8190629482269287, -0.08706089854240417, -0.33877071738243103, 1.7153829336166382, 0.10267328470945358, 0.15871424973011017, -0.032887544482946396, 0.0015141622861847281, 0.25039443373680115, 0.28238344192504883, 0.04692242294549942, 0.9671884775161743, 0.3591698408126831, -0.969802737236023, 0.758507251739502, -0.1072952076792717, -0.8639174699783325, -0.45997217297554016, 1.071258544921875, 0.5271866917610168, -0.058501049876213074, -0.6374729871749878, -0.5727595090866089, 0.7834329009056091, -0.7549049258232117, 0.7689728736877441, 0.9893330335617065, -1.9390103816986084, 1.2612576484680176, -0.02543984353542328, 2.3123786449432373, -0.16150052845478058, -0.22688646614551544, -1.3312488794326782, -1.989346981048584, -2.3470349311828613, -1.2027955055236816, -1.3897507190704346, 1.013801097869873, -0.6626316905021667, 0.15536780655384064, 1.236828327178955, -0.051286786794662476, 0.6863105297088623, 1.110920786857605, -0.8306609392166138, -0.5384237170219421, -0.36341404914855957, 1.4164793491363525, -0.1711769700050354, 0.8525975346565247, -1.765395164489746, -0.25025075674057007, 0.3901060223579407, -1.3168953657150269, 1.5667016506195068, -0.5202155709266663, -1.670764684677124, -1.0251405239105225, -0.8782945871353149, -0.08708319813013077, 0.815527617931366, -0.8851404190063477, -0.5735394358634949, -0.6719465255737305, 0.02399441972374916, 0.1690269261598587, 0.22586555778980255, 0.9398393630981445, -0.09402600675821304, 1.2806354761123657, -0.22308634221553802, -0.1883421093225479, 2.17585825920105, 1.1384245157241821, 1.3830955028533936, -0.5929419994354248, 1.5198304653167725, -1.4615507125854492, -0.7791024446487427, -2.244034767150879, -1.1297221183776855, -0.5954902768135071, -1.3290799856185913, 1.5569286346435547, 0.4067241847515106, -0.5206724405288696, 0.21441704034805298, -0.09902804344892502, -0.8722795248031616, 1.872721791267395, 0.4631751775741577, -0.018519308418035507, 1.1286730766296387, 1.0488579273223877, 0.9236389994621277, -1.2819377183914185, -1.0524860620498657, 0.38638219237327576, -0.2595907747745514, 1.9801044464111328, -0.2053932249546051, 0.7802750468254089, 1.4446632862091064, -0.5690616369247437, -1.2789487838745117, 0.5902032852172852, -0.650368869304657, 0.38614988327026367, 1.5939847230911255, 0.9864494800567627, 0.6986870765686035, 0.6808732151985168, -0.693098783493042, 1.7065515518188477, -0.650737464427948, -1.3981938362121582, -1.186618447303772, 0.9064636826515198, -1.6635701656341553, -0.6756727695465088, 0.8458297252655029, -0.10062606632709503, 0.19747225940227509, 0.09565827995538712, -0.9931228160858154, -0.40150439739227295, 0.5337340235710144, 1.585984230041504, -0.5198736190795898, 1.0801630020141602, 0.048836737871170044, -0.5282350778579712, -1.5099055767059326, -0.7153972387313843, -0.6149089336395264, 0.45050761103630066, -0.15933646261692047, -0.43569415807724, 0.6299996972084045, -0.7988706231117249, 0.26497429609298706, 0.5878692269325256, -1.253129243850708, 1.0107073783874512, 0.09162729978561401, 0.26963913440704346, -0.21031171083450317, -0.19744592905044556, -0.5341642498970032, 0.503576934337616, -1.503279685974121, 1.1213552951812744, 0.49013790488243103, 0.6247184872627258, -0.4414929449558258, 0.8958348035812378, -0.4115862250328064, -0.9078325033187866, 0.12022978812456131, 0.374705046415329, 0.9936545491218567, 0.9318457841873169, -1.530791163444519, 0.9129258990287781, 0.8871380090713501, -0.26432281732559204, 1.6609560251235962, -0.1695905476808548, 0.2893088161945343, 1.3457435369491577, -0.7998853325843811, 1.7079309225082397, -0.4609110355377197, 0.33274877071380615, 2.5485687255859375, -1.2827237844467163, 1.7975304126739502, 0.04896550625562668, -0.2814289629459381, -0.22210805118083954, -0.32600077986717224, -1.5306144952774048, -1.1313918828964233, 1.2602767944335938], [0.43790745735168457, 1.5679301023483276, 0.536531925201416, 0.7842811346054077, 0.7365493774414062, -0.2243914008140564, -0.8150104880332947, -1.26218581199646, -1.237833023071289, 1.7348434925079346, 1.1499934196472168, 0.9163199067115784, -1.364654302597046, 0.18497243523597717, 0.12978966534137726, -0.4039817750453949, -0.5937106013298035, -0.05188707634806633, 1.0571837425231934, -0.09564858675003052, 1.4880845546722412, 0.16632971167564392, 0.3421919643878937, 1.4339611530303955, 0.5179017782211304, -0.4952797591686249, 0.407418429851532, -0.7113341689109802, 0.5212907791137695, -1.407741904258728, -0.6566196084022522, 0.02819257415831089, -0.779270350933075, -2.1924736499786377, -9.290796151617542e-05, -1.4073774814605713, 0.2135850489139557, 0.24173220992088318, -0.2578597068786621, 0.8242326378822327, -0.20370441675186157, -1.2641750574111938, 0.31395623087882996, -0.05555237829685211, -0.33090516924858093, 0.40324220061302185, -0.577759861946106, -0.05778373405337334, -2.187033176422119, -1.2578845024108887, -0.40014898777008057, -0.5325465202331543, 0.26343515515327454, -0.46191030740737915, 2.278144359588623, -0.21240422129631042, -0.9262500405311584, 1.5285323858261108, -0.7992755770683289, 0.8550788760185242, -2.631286382675171, 0.9325428605079651, -0.9926683902740479, -0.698602557182312, -0.15546683967113495, -1.2183418273925781, 1.872012734413147, -0.22388499975204468, 0.7183324098587036, 1.8397008180618286, -0.9213116765022278, -0.33186033368110657, -0.01087296474725008, -0.4663371443748474, -1.324004888534546, -0.12637130916118622, 1.3695672750473022, 0.0018243344966322184, -0.06654614955186844, -0.9560292363166809, -0.08444283902645111, -1.1366502046585083, 0.6420584917068481, 0.40092626214027405, 0.045741334557533264, 1.0118587017059326, -0.597948431968689, -0.40813568234443665, -0.5088850259780884, 0.5860981941223145, -0.7283251285552979, -0.13849541544914246, -1.3898255825042725, -0.22527951002120972, -0.07692118734121323, -1.0291110277175903, -0.08716358244419098, -1.2808971405029297, 0.652463972568512, 0.12413843721151352, 0.5267636179924011, -1.3869478702545166, 2.210904359817505, -1.3091424703598022, -0.17300128936767578, 0.005997190717607737, -0.009443613700568676, -1.337947130203247, 0.38660502433776855, 0.4625335931777954, 0.13429245352745056, -0.12282203882932663, 0.05217724293470383, 0.17587514221668243, -1.241174340248108, 0.08631916344165802, 0.7489863634109497, 0.5431352257728577, 1.200427770614624, -0.21481339633464813, -0.4263201951980591, -1.707515835762024, -0.5459050536155701, 0.2970133125782013, -1.0593374967575073, 0.4762735962867737, -1.099524974822998, 0.021727925166487694, -0.887055516242981, 0.39965981245040894, 0.6084951162338257, -1.859029769897461, -1.2515795230865479, -0.15248984098434448, -0.6283735036849976, 2.5172171592712402, -0.4169318377971649, -0.2940220832824707, 0.2121993750333786, -0.30326610803604126, 1.2422682046890259, -1.0904104709625244, -0.7389141917228699, -1.4775482416152954, -1.097676157951355, -0.43330496549606323, 1.7038317918777466, 0.925541341304779, 0.6700223684310913, -0.7574889659881592, -0.816384494304657, 0.23998363316059113, -1.0052067041397095, 0.6947823762893677, 0.27166903018951416, -0.1576690524816513, -0.37254273891448975, 0.45663294196128845, -1.2731281518936157, 2.257133960723877, -0.3151194155216217, 2.1251702308654785, -0.25557222962379456, -0.8361424207687378, 0.8142926096916199, -0.30644577741622925, -1.4642966985702515, 0.011203985661268234, 0.6085339784622192, 1.8414424657821655, -0.3092886507511139, 0.5561598539352417, -0.40217098593711853, 0.5627238750457764, -1.1815723180770874, 0.46643078327178955, -0.810893177986145, -1.1129487752914429, -0.26971372961997986, -0.47316837310791016, -1.2289656400680542, -1.2436070442199707, -1.5679478645324707, 1.5342254638671875, -0.2398575395345688, -0.6306816935539246, -0.8225563764572144, 0.4742526710033417, -0.6075950264930725, -1.3387645483016968, 0.557235836982727, -1.0376708507537842, 0.13152019679546356, -0.2354845255613327, -0.9841665029525757, 0.8446106910705566, 0.9261769652366638, 0.43077078461647034, 0.6488500833511353, 0.34526965022087097, -0.27906301617622375, 0.13996268808841705, -0.8046308159828186, 0.8080390691757202, 0.24689891934394836, 2.134467840194702, 1.418756127357483, 0.7911406755447388, -1.7864928245544434, 0.2811053395271301, 0.29205021262168884, -0.6000596284866333, 0.3887987732887268, -1.4441484212875366, 0.17528757452964783, -0.7264362573623657, -0.7911137342453003, 0.5873115062713623, 0.27993714809417725, -0.06512485444545746, 0.7440053224563599, -0.018363509327173233, -1.2405586242675781, -0.4532776474952698, 0.09428517520427704, 1.5822248458862305, 2.491089105606079, 1.6840057373046875, -0.5699859857559204, 1.0183522701263428, 0.8881202340126038, -0.4384089410305023, 0.9788669347763062, 0.909041702747345, -0.8793814778327942, -0.8465709090232849, 1.566667914390564, 1.3422274589538574, -1.1187517642974854, -2.8131816387176514, 0.6516337394714355, -0.7070314288139343, 0.521801233291626, -0.28606078028678894, -0.6525978446006775, -0.2227136194705963, -0.1924619823694229, 2.2206075191497803, -0.7397668957710266, -1.0014065504074097, -0.3169915974140167, -1.822408676147461, -0.6134752035140991, 0.34111347794532776, -0.9309418201446533, -0.16905581951141357, 0.5253632664680481, -0.47408416867256165, 0.25434133410453796, 0.27326279878616333, 0.6978073120117188, 1.0939675569534302, -0.0759098157286644, 0.6187334060668945, 0.3377223610877991, 0.744835376739502, 0.9497461915016174, -0.3264373242855072, -1.5866048336029053, 0.8673356175422668, -0.04130018502473831, 0.4441024363040924, 0.7118986248970032, 0.8064670562744141, 0.10846316814422607, 0.48487231135368347, 0.554881751537323, 1.5743608474731445, 1.021147608757019, 0.0868246927857399, 1.2026808261871338, 0.415164977312088, -0.0012306704884395003, 0.33284521102905273, 0.3627426326274872, -0.6308352947235107, -1.5855387449264526, -1.5895261764526367, -0.9167627096176147, 1.4647923707962036, 0.6314200758934021, -0.7305038571357727, 1.3931002616882324, 0.4557817578315735, -1.208731770515442, 1.2401680946350098, -0.7119591236114502, 0.8608524203300476, 0.6835821270942688, -0.986635148525238], [0.4042832851409912, -0.0994105115532875, -2.193939447402954, 0.16191552579402924, 0.9947343468666077, -0.9535946846008301, -0.08352767676115036, 0.42214763164520264, -0.24036960303783417, 0.6926632523536682, 0.3101096749305725, 1.4501042366027832, 1.715491771697998, 2.410417079925537, 1.4981820583343506, 2.1539673805236816, -1.9699782133102417, 0.6660344004631042, -1.1563881635665894, -0.18130382895469666, -0.44659310579299927, 1.756769061088562, 0.7963460683822632, -2.4514758586883545, -0.8715903759002686, -0.954716145992279, 1.2862582206726074, 0.13018076121807098, -2.05352520942688, 0.734136164188385, -0.5019310116767883, 1.0267186164855957, -0.7045841217041016, -0.3783980906009674, 0.07553070038557053, -1.5632400512695312, -0.5530015826225281, -1.0435004234313965, -1.255996823310852, 2.2707977294921875, 0.6611595749855042, 0.7531646490097046, 0.8111194968223572, -0.6013814806938171, 1.2395514249801636, 0.15160255134105682, -1.0008561611175537, -0.2551898658275604, 0.8092975616455078, 0.492710143327713, -0.7877140641212463, 1.2684845924377441, 0.2388051450252533, -0.7227203249931335, -0.31070229411125183, -0.23936767876148224, 1.9726864099502563, -0.49875298142433167, -1.3124233484268188, 1.283121943473816, -1.149069905281067, 0.8245584964752197, -0.4950973689556122, 1.5116466283798218, -0.5567916035652161, -0.15168578922748566, 0.15308614075183868, -0.8653342723846436, -0.31571829319000244, 0.7148305773735046, -0.8994499444961548, 0.8024782538414001, -2.4544517993927, 0.48841625452041626, 0.47115474939346313, -0.36817485094070435, 0.408027321100235, -0.3180987536907196, -0.419235497713089, -0.5609071254730225, -0.169053316116333, -1.1417936086654663, -0.4594208598136902, -0.6886415481567383, 0.2925662100315094, 1.6212971210479736, -0.8344891667366028, -1.267235517501831, 1.998102068901062, 0.36528733372688293, 0.9366747140884399, -0.18678802251815796, 1.106433391571045, 0.013953875750303268, 0.926349937915802, -0.9468737840652466, 0.6833379864692688, -0.5007328987121582, 0.613010048866272, 0.08115355670452118, 1.5341564416885376, -0.016360271722078323, 0.07803880423307419, 0.16476468741893768, 0.9723313450813293, -1.298435926437378, -0.7448440194129944, 1.7452595233917236, 0.4061797261238098, 0.6518672108650208, 1.4867420196533203, 0.5343943238258362, -0.6978287100791931, 0.17911982536315918, -1.1389282941818237, 0.5270056128501892, -0.28410232067108154, 1.8298147916793823, -0.4297075569629669, -0.13286976516246796, 0.330849826335907, -0.40815460681915283, -0.8559203743934631, 0.08294501155614853, -0.19751019775867462, 1.2325879335403442, 0.658997118473053, 1.420163869857788, 0.18704164028167725, -0.3895086646080017, 0.6996434330940247, 0.15242110192775726, -1.3288763761520386, -1.4001555442810059, -0.915977954864502, 0.8011795878410339, -0.19344520568847656, -0.4999108612537384, -0.5097007751464844, -0.9599102735519409, -0.6303621530532837, -0.0420270636677742, 0.08152040839195251, 1.9122542142868042, -2.8718724250793457, 0.09765065461397171, 0.3178744316101074, -1.2619982957839966, -0.4554687440395355, -0.45467689633369446, 0.9192954897880554, 0.9182444214820862, 0.0437454991042614, 0.37843960523605347, -0.1654503047466278, -0.15730425715446472, 0.48428308963775635, 0.11768288165330887, -1.1686757802963257, 0.6541730761528015, -1.1780784130096436, -0.8486701846122742, 0.6472417712211609, 0.41912585496902466, -0.11039754003286362, -1.2213118076324463, -1.175218939781189, 0.4754217565059662, 0.8082408308982849, 0.47481754422187805, 0.9483540654182434, 0.36503663659095764, 0.49271002411842346, 0.31607961654663086, -0.881502628326416, -0.8815389275550842, 0.11348989605903625, -0.3418773412704468, 0.4073465168476105, -0.4603462517261505, -0.09186733514070511, -0.7821739912033081, 0.4256359338760376, -0.3551742732524872, 0.5242582559585571, -1.4041424989700317, -0.8439657092094421, 0.2147650420665741, -0.9258025884628296, 0.45824670791625977, 0.43373098969459534, -0.9024405479431152, -1.4824496507644653, 0.2715663015842438, -0.3906761705875397, -0.7671983242034912, -0.3689860999584198, -0.3455086052417755, 0.24644476175308228, 0.5031618475914001, 0.7747778296470642, -0.29991796612739563, 0.41134360432624817, -0.6059593558311462, 1.0294272899627686, -0.38601556420326233, 0.49348318576812744, -0.030924297869205475, 0.3304361402988434, 1.246368408203125, -1.3718465566635132, 0.10264835506677628, -0.9985305666923523, 0.2976352572441101, -2.0377845764160156, 1.0637720823287964, -1.3033647537231445, -1.1270859241485596, 0.4287135899066925, -1.0491188764572144, 1.4264782667160034, -1.1803008317947388, -1.7146341800689697, 1.452151894569397, -0.01048924122005701, -0.5380922555923462, 0.33673718571662903, 1.5557823181152344, 1.7409813404083252, 0.2729410231113434, -1.6248695850372314, 1.212201476097107, 1.240134358406067, 0.8854650259017944, 0.12198453396558762, -1.7192775011062622, 0.8068536520004272, -2.2528750896453857, -1.1823275089263916, -1.1045416593551636, 0.5575203895568848, 1.832038402557373, 1.17886483669281, 2.1396877765655518, -0.7414425611495972, 1.1714863777160645, -0.9008634686470032, -0.8920905590057373, -1.7404130697250366, -0.538396954536438, -0.029749440029263496, 0.21587783098220825, 2.060101270675659, 0.27673014998435974, -0.5853962898254395, 0.1025707945227623, 0.4827096462249756, -0.22105714678764343, -0.5146435499191284, 0.0968400239944458, -0.1222052201628685, -0.5439648628234863, -0.39545199275016785, 0.24899572134017944, 2.7966256141662598, 0.02729206718504429, 0.06728993356227875, 0.5952745079994202, 1.0414077043533325, 1.8805195093154907, 0.23688293993473053, -0.905829131603241, 2.2143847942352295, 0.7644758820533752, -1.5968009233474731, -0.28600603342056274, 0.8519396781921387, 0.9654734134674072, -0.4128321409225464, 0.7775760889053345, -1.5064963102340698, 0.3653256297111511, 0.4221743047237396, 0.09881658852100372, -0.3646407425403595, 0.17087674140930176, -1.3114453554153442, -0.43563076853752136, -0.1907123178243637, 0.5430322289466858, -0.14381437003612518, 0.7083649635314941, -0.8801275491714478, -0.17164328694343567, 2.1031458377838135, -0.5104164481163025, 1.4806764125823975, 0.055040907114744186, -1.013324499130249, 1.0347167253494263], [-0.7422929406166077, 0.8775540590286255, -1.127771258354187, -1.0194803476333618, -1.267737627029419, -0.18262159824371338, 0.2893628180027008, 0.6185925006866455, 0.6519623398780823, 1.323850393295288, -0.5882946848869324, -0.16494424641132355, -0.8544935584068298, 0.8243910670280457, 0.6274433135986328, -0.050284773111343384, -0.0010219302494078875, -0.424272358417511, -0.6569504737854004, 0.4506629407405853, 2.4289333820343018, 1.250022530555725, -1.444205403327942, -0.47249072790145874, -1.0576574802398682, 0.3365190923213959, 0.8016084432601929, 1.4204589128494263, -0.9624655246734619, -1.9352587461471558, 2.4109947681427, 0.0887182429432869, -0.27403268218040466, 0.1269708126783371, -1.594690203666687, -0.9423770904541016, 0.1949404925107956, -0.20044097304344177, 0.04612492024898529, 0.01473464909940958, -1.6504606008529663, -0.18666625022888184, 1.0563907623291016, 0.10394520312547684, -0.4762190282344818, -1.0835084915161133, 1.2638919353485107, 1.2166138887405396, 0.9100511074066162, 0.3406328558921814, -2.1170594692230225, -0.14040027558803558, 0.5388339757919312, -0.05275863781571388, -0.024111641570925713, 0.9216820597648621, -0.997162401676178, 0.7008610367774963, -0.5027410387992859, 1.2654104232788086, -0.10131648182868958, -1.3310540914535522, -0.19520577788352966, 0.4925205409526825, 0.28364771604537964, -1.0364046096801758, 0.7794317007064819, -0.24724756181240082, -0.5334128141403198, 1.8907982110977173, 0.36726099252700806, -0.9028955698013306, -0.7686325907707214, -1.2720752954483032, 0.47787660360336304, -0.08054377138614655, 0.5158798098564148, 0.5394856333732605, -0.035206880420446396, 0.024727407842874527, -1.2863465547561646, 0.8156433701515198, -0.8140807747840881, 0.9400792717933655, -0.11934655159711838, -0.405527263879776, -1.5827354192733765, -2.050626277923584, 1.1320581436157227, 0.17181849479675293, 0.7856962084770203, 0.5159167051315308, 0.7288742661476135, 1.8039934635162354, 0.21779362857341766, 0.13472715020179749, -2.0733001232147217, 0.7048132419586182, -1.1914585828781128, 0.6912432312965393, -0.9126099944114685, 0.3467811644077301, -0.2743977904319763, -0.7819196581840515, 0.6122295260429382, -0.8769237399101257, -1.0063728094100952, 0.2791873514652252, 0.7285305857658386, -0.16169226169586182, -2.4873476028442383, 0.9154718518257141, -1.6789590120315552, -0.1381920427083969, 1.1909370422363281, 1.0826281309127808, 0.16170498728752136, 0.7221053242683411, 1.713213562965393, -0.7613810896873474, -0.7650812268257141, 0.41606396436691284, -0.4936772584915161, 0.48258060216903687, -0.08006177097558975, -0.8957528471946716, 1.704771876335144, 0.42231228947639465, -0.209150493144989, 0.7973619699478149, -0.12082489579916, -1.150399088859558, 0.6242095232009888, -1.0219173431396484, -0.5331392288208008, -0.8148007392883301, 1.7585937976837158, 0.02149156853556633, 0.2496490329504013, 1.6172643899917603, -0.5453439950942993, -0.9063735008239746, 1.6136897802352905, 0.05312270298600197, -0.45117175579071045, -1.0335581302642822, 1.5163350105285645, 0.688713014125824, -0.6883318424224854, -0.938064694404602, -1.2107291221618652, 0.2434619814157486, -1.0204334259033203, 1.342289686203003, 2.363361358642578, 0.29250800609588623, -0.9464211463928223, -1.0052921772003174, -0.768796443939209, 0.8404027819633484, -0.5857287049293518, -0.774483323097229, -0.39771202206611633, -0.6327853798866272, 0.5834477543830872, -0.4940250515937805, 0.6895827054977417, -1.1326642036437988, -1.8881620168685913, 0.058415450155735016, 0.44840332865715027, 1.7659560441970825, -0.9020563960075378, 0.2536897659301758, -0.3612564206123352, -0.08938171714544296, -1.5395045280456543, -1.2783070802688599, -0.8495900630950928, -0.3576348125934601, -0.11330486834049225, 1.1231070756912231, -0.5249866843223572, -0.12753571569919586, -1.6603777408599854, -0.4639928638935089, 0.861270010471344, 0.021103128790855408, -1.3038628101348877, -0.6032576560974121, 0.14362937211990356, -0.6432276964187622, -1.1939153671264648, -0.1275145411491394, -0.5939440727233887, -0.8260950446128845, 0.5945062041282654, 0.15996284782886505, 0.5088896751403809, 0.5130250453948975, -0.22407004237174988, -0.12734240293502808, -0.9959793090820312, -1.5307691097259521, 2.9355335235595703, 0.22315707802772522, -0.3677786886692047, 0.5097876191139221, -0.41742053627967834, -0.5133457183837891, 0.21034695208072662, 0.5463960766792297, 0.12507513165473938, -2.3481967449188232, 0.06055448576807976, 1.8190820217132568, -0.5778260231018066, 0.18246470391750336, -0.8700085878372192, -0.06505878269672394, 1.8811836242675781, 1.0771992206573486, 0.29060184955596924, 2.1841118335723877, -0.4624217748641968, -1.0871191024780273, 1.1063835620880127, 0.12440751492977142, 1.5870460271835327, -1.6179908514022827, -0.600730836391449, -0.21861311793327332, 0.0870710164308548, -1.0457030534744263, -0.7852121591567993, -0.6400846242904663, 2.5937607288360596, 3.584993839263916, 2.0531651973724365, -1.2197370529174805, 0.2774239480495453, 0.5835038423538208, 1.0395381450653076, -0.5082026124000549, 1.752308964729309, 1.074445366859436, -0.9751614928245544, -0.2134130448102951, -0.8583329319953918, -0.20071610808372498, -0.7724676728248596, -0.6355926394462585, -1.5192023515701294, -0.16581317782402039, -0.6744145154953003, -1.7655795812606812, 0.6599634289741516, 0.6669192910194397, 1.922713279724121, 0.7865065336227417, -0.8178918957710266, -0.04119513928890228, -0.17654302716255188, 0.1589319407939911, -0.5693472027778625, 0.6073152422904968, 0.2995067238807678, -0.9741413593292236, -0.10256578773260117, -1.3972166776657104, -1.6734308004379272, 0.3205820620059967, -0.912246584892273, 1.4846950769424438, 1.2782572507858276, -1.8807148933410645, -1.0642099380493164, 0.3790619969367981, -0.5013304948806763, 0.20344243943691254, 0.13430434465408325, 0.10515191406011581, -1.2900139093399048, 0.5869589447975159, -0.41156718134880066, 0.666904628276825, 0.5096496343612671, 0.21918386220932007, -1.3009759187698364, -0.7920945286750793, -0.9148834943771362, -0.537601888179779, -2.012021780014038, 0.7644027471542358, -0.9235211610794067, 0.6727570295333862, 0.1692950576543808, 0.8869728446006775, 1.0288689136505127, -0.08225487917661667], [0.5608816146850586, -0.56655353307724, -1.0198264122009277, -0.32411694526672363, 1.2887012958526611, 0.45067089796066284, 1.7982348203659058, 0.40592440962791443, -1.0566012859344482, -0.9663071632385254, -1.328285813331604, 0.6953445076942444, 0.15431976318359375, -1.0759485960006714, 0.38645681738853455, -0.5114383697509766, -0.47514691948890686, -1.3936539888381958, 0.6444758176803589, -2.783891201019287, 0.19387079775333405, 0.45485395193099976, -0.7240920066833496, -0.10175100713968277, -0.7734364867210388, 2.093473196029663, 0.4050653278827667, 0.5845321416854858, 1.5399401187896729, -0.3536182641983032, -1.24908447265625, 0.13409669697284698, -0.5811552405357361, -0.7725316286087036, -0.18900153040885925, 0.9100581407546997, -1.3981670141220093, -0.33394336700439453, 2.0067062377929688, 0.5742225050926208, -0.21573662757873535, -0.6292316913604736, -0.5781566500663757, -0.3462487459182739, 1.255508542060852, 0.8876554369926453, 0.5318299531936646, -1.1348539590835571, -0.12328364700078964, -0.9428002834320068, 0.4271739721298218, -0.2027682512998581, 1.3092014789581299, -0.8054393529891968, 0.09626087546348572, -1.0980827808380127, 0.27200016379356384, 0.7975304126739502, 0.12129885703325272, 0.19457174837589264, 0.3591078817844391, -0.02951807528734207, -0.26620808243751526, 0.01608678326010704, 0.13388508558273315, -0.6686088442802429, -1.4533318281173706, -0.19572864472866058, -0.7213045954704285, -0.9197257161140442, 0.5027555823326111, -1.2597243785858154, 2.2258493900299072, 0.2803572416305542, 1.8694533109664917, 1.5214911699295044, 0.25552424788475037, 0.8292901515960693, -0.38570353388786316, -1.5489989519119263, -0.5328316688537598, 0.5428902506828308, -1.1513234376907349, 1.1277625560760498, 1.069213628768921, -1.0177158117294312, 0.6075735092163086, -1.2401782274246216, 0.22181986272335052, 0.12212488800287247, 2.4153287410736084, -0.8606523871421814, -0.4659002423286438, 1.999214768409729, -1.2910332679748535, -1.943024754524231, 0.5754488706588745, 1.0499616861343384, 0.06577767431735992, -0.7600447535514832, -1.195831060409546, 0.3570968806743622, 0.29526886343955994, -1.0047175884246826, 0.9314813613891602, 0.4246009290218353, -1.112560749053955, -0.638502299785614, -2.693542003631592, -1.2213225364685059, 1.3597122430801392, -1.156509518623352, -0.06829284876585007, -1.1458669900894165, 2.036515235900879, 1.0509408712387085, 0.33801981806755066, -0.5462597608566284, -0.8401133418083191, -0.27554967999458313, 0.04503892362117767, -0.0743439570069313, -0.021614840254187584, -2.513792037963867, 0.14443880319595337, -0.2641042470932007, -0.6058708429336548, 0.038144052028656006, 1.2463510036468506, -1.2281666994094849, -0.24114647507667542, 0.02775581181049347, 1.3431602716445923, -0.8949857950210571, 2.6307523250579834, -0.5081065893173218, -0.6603085398674011, -0.4408141076564789, -1.616105079650879, -0.23474283516407013, -0.17121130228042603, -1.1468505859375, -0.798562228679657, 1.298708438873291, 1.3504003286361694, 0.3874572217464447, -0.4736306965351105, -2.125079393386841, 0.323748379945755, -0.49396467208862305, 1.6670751571655273, 0.1370609998703003, 0.06913008540868759, -0.36028853058815, 0.9851264357566833, -1.7042783498764038, 1.051372766494751, -1.8190103769302368, -1.9079290628433228, -1.5275986194610596, 0.916429340839386, 1.3626126050949097, 0.35921746492385864, 0.12883108854293823, -1.3296147584915161, -1.5311169624328613, 0.5488252639770508, 0.3875559866428375, -1.0002827644348145, -2.3591949939727783, 1.2413678169250488, 0.3124001622200012, -0.8911981582641602, -1.1424219608306885, -0.05127455294132233, 0.7156319618225098, 0.5439456105232239, -0.2912910580635071, -0.3343721032142639, 0.775372326374054, -2.031125545501709, 0.12577509880065918, 0.25564974546432495, 0.9252201318740845, 1.608689785003662, 1.9920918941497803, -0.45176002383232117, 0.0365600623190403, -1.694490671157837, -0.3218836486339569, 0.9556052684783936, -0.06973515450954437, 0.9033012390136719, 0.01373044028878212, 1.1843199729919434, 0.21763916313648224, -0.5804113149642944, 0.33086061477661133, -0.34883829951286316, -0.9782190918922424, -0.2593671679496765, 0.6039292812347412, -0.2096635103225708, -1.306898832321167, 1.3583840131759644, -0.44530126452445984, 0.23879097402095795, -2.490361452102661, -0.02546454593539238, 0.1908433884382248, -2.285104274749756, 2.0120036602020264, -1.5016077756881714, -2.9423413276672363, -1.785129427909851, -0.9681679606437683, 1.2527267932891846, -0.38652849197387695, 0.13200822472572327, -0.8214184641838074, -0.23442502319812775, -0.3105168044567108, -1.5768858194351196, -0.10108950734138489, -0.7477129101753235, 0.3104150891304016, 0.7359935641288757, -0.6276253461837769, 0.9429548978805542, -0.08298491686582565, 0.28428155183792114, 0.285525918006897, 0.8205974698066711, 0.8918286561965942, -1.253773808479309, 0.8644250631332397, 0.435115247964859, 0.07379639893770218, -1.0127485990524292, 0.01800917647778988, -0.10677362978458405, -0.2968290448188782, -1.5516587495803833, -1.280196189880371, 0.3331420123577118, -1.3990463018417358, -0.6411288380622864, -0.5788871645927429, 1.1335822343826294, -0.6435521841049194, 0.16687525808811188, 0.6325035691261292, 0.37424781918525696, -0.07304496318101883, -1.0121724605560303, 0.8461349010467529, -1.3187243938446045, -0.2873597741127014, 0.4426727294921875, -0.5987730026245117, 0.08903960138559341, 0.5709367990493774, 0.5406990647315979, -0.3252648413181305, 0.07354838401079178, -0.3198871314525604, 1.8052064180374146, -0.6308163404464722, -0.7646650671958923, 2.5735256671905518, -0.3625795543193817, -0.897384524345398, 0.4867551028728485, -0.09825985133647919, 0.8632354736328125, -0.000664179096929729, 1.4195806980133057, -0.7268773913383484, 0.12241137027740479, -0.32380586862564087, -0.08236084133386612, -1.540450930595398, -0.24209466576576233, 0.2289290428161621, 0.1419258713722229, -0.16066230833530426, 0.7639878392219543, 0.25584152340888977, -0.48278942704200745, -0.827070415019989, 3.1804211139678955, -2.0682473182678223, -0.6153672933578491, -0.2640531063079834, 0.1470516324043274, 1.1673693656921387, -0.6313159465789795, -0.4726593494415283, -0.5648536086082458, 1.0275846719741821], [-1.3792386054992676, -1.5116195678710938, -0.9340974688529968, 0.6428205966949463, -0.9704793095588684, -1.0328129529953003, -0.21628262102603912, 1.0378843545913696, 1.5383989810943604, 1.9214000701904297, -1.5432648658752441, 0.1517374962568283, -0.8706948161125183, 0.9595737457275391, 0.7572264671325684, 0.2764773368835449, 0.11597380042076111, -0.7989601492881775, 0.807834804058075, -0.18696236610412598, -0.4145970940589905, -1.0358332395553589, 0.46441394090652466, -0.30989059805870056, 0.6692160964012146, -0.26576554775238037, -0.0951940268278122, 0.19898417592048645, 0.3605136573314667, 0.7656866312026978, 0.23720020055770874, 1.4714865684509277, -1.715504765510559, -1.3746757507324219, 1.6995316743850708, 1.339072585105896, 0.9476132392883301, -1.2058347463607788, -1.074524998664856, -0.45897066593170166, 1.1340121030807495, -0.6581047773361206, 0.327316552400589, -1.4250577688217163, -0.7409846186637878, 0.0018662001239135861, 0.5920813083648682, -0.8325302004814148, -0.6757733821868896, -2.4390265941619873, -0.42537277936935425, 0.5460905432701111, 0.9478591084480286, 0.4790527820587158, -0.3524355888366699, 0.6177014708518982, 0.3224066197872162, -0.9522129893302917, -0.7758025527000427, 0.6544923186302185, -0.2691902816295624, 0.8732175230979919, 2.1421892642974854, 0.12837444245815277, 2.281548500061035, -0.8927555084228516, 2.669135093688965, -0.11792708188295364, 0.13361036777496338, 0.1591818630695343, 0.951137125492096, -0.25643205642700195, -1.1469793319702148, -0.40778565406799316, -0.5423908829689026, -0.45986372232437134, -1.3625662326812744, -0.1307884156703949, 0.36626574397087097, -1.6634124517440796, -1.0466853380203247, -0.4910604953765869, 1.0993150472640991, -0.3482016324996948, -0.9897593259811401, -0.03545738384127617, -0.5032524466514587, 0.154892697930336, 1.4982576370239258, -0.410200834274292, 0.4889640808105469, -0.09545300900936127, -1.4207689762115479, 0.515700101852417, 1.5177925825119019, -0.43025580048561096, 1.058275580406189, -0.47821271419525146, 1.358593225479126, 0.8713352680206299, -2.526851177215576, -0.8008816838264465, -2.129157781600952, 1.984634518623352, 1.9109910726547241, -0.2341151237487793, -0.592823326587677, -1.8406022787094116, -0.10939870029687881, 0.3230910301208496, 0.8629297018051147, 0.6394241452217102, -0.7875989079475403, 0.19011861085891724, 0.8827396631240845, 1.1643216609954834, -1.0334391593933105, -0.6687933206558228, 0.14635471999645233, -0.44262707233428955, 0.2385227382183075, -0.5386795997619629, 0.26428350806236267, -0.2781504690647125, -1.188451886177063, 0.3430679440498352, 0.39931419491767883, -1.3677096366882324, 0.24988995492458344, -0.021869037300348282, -0.6657992005348206, -0.01695890910923481, -0.8307489156723022, 1.2265727519989014, 0.3429107666015625, -0.8307218551635742, -0.06959118694067001, 0.08053933084011078, 1.0988236665725708, -2.0587637424468994, 0.2799546420574188, -2.317479133605957, -0.30928972363471985, -0.6709238290786743, -2.5238184928894043, -0.6472582221031189, 0.13011671602725983, 0.10933669656515121, -0.8784310221672058, -0.7003933191299438, 0.6252686977386475, -1.362513542175293, 0.45527565479278564, -0.08756930381059647, -0.06972305476665497, 2.2637932300567627, 0.4144440293312073, 0.7659869194030762, -0.0035047500859946012, 1.0424048900604248, -1.3333344459533691, -1.6326993703842163, 0.6785702109336853, 0.2604031562805176, -0.5568237900733948, -1.4952392578125, 1.864436149597168, 0.8492434024810791, 0.7936921715736389, -0.267270565032959, -0.23363493382930756, -2.2976155281066895, 0.3736548125743866, -0.21866808831691742, -1.05472731590271, -0.47857344150543213, -0.35961735248565674, 0.7381726503372192, -1.1929302215576172, 2.1987428665161133, -1.2774003744125366, -0.2560185492038727, 0.4743237793445587, 0.030025862157344818, 1.5439741611480713, -0.7574579119682312, 0.4092719256877899, -0.42981240153312683, 0.13100892305374146, 0.3009986877441406, 0.3181217610836029, 0.21304373443126678, 0.12729261815547943, 2.159761905670166, 1.8847599029541016, 1.0439069271087646, 0.15022048354148865, -0.9121199250221252, 0.5986389517784119, -0.2868863642215729, 0.07748934626579285, 0.4190601110458374, 0.045336708426475525, -0.213875412940979, -0.19150054454803467, 0.3249783217906952, 1.3054629564285278, -0.4640825092792511, -0.7213593125343323, -0.53707355260849, 0.6518223881721497, 0.6526743769645691, 0.623157262802124, -1.8983988761901855, -0.7637448310852051, -0.354919970035553, -0.6046085357666016, 1.2973484992980957, -0.20152586698532104, 1.4238262176513672, 0.920685887336731, 1.3631923198699951, -0.2335359752178192, 1.9171812534332275, 1.1723124980926514, -0.16185317933559418, -0.8679604530334473, -1.9115948677062988, -1.0927152633666992, -0.608680248260498, 1.3797575235366821, -0.7117887139320374, 0.6980078816413879, -0.34612175822257996, 0.20062828063964844, 0.7707989811897278, -0.027684150263667107, 1.2924922704696655, -1.5346224308013916, -0.6033391952514648, 0.1688864380121231, 1.0460938215255737, 0.752709150314331, -0.6924112439155579, -0.2800943851470947, 0.10247202962636948, -2.350036859512329, 0.07904475927352905, -0.3970588743686676, -1.0636166334152222, -0.4928046464920044, -1.08672297000885, -0.5403863191604614, 0.32963213324546814, -0.004725523293018341, -0.33180198073387146, 0.17282165586948395, -0.6316126585006714, -0.2460094690322876, 2.2186827659606934, -0.3616894781589508, 0.6785950064659119, 0.4561680257320404, -1.3101133108139038, -0.6792042851448059, -0.17110179364681244, 2.1716291904449463, 0.042962606996297836, -0.02886640653014183, 0.5200042128562927, 0.06376995146274567, -0.8459268808364868, 0.631312370300293, 0.9839785099029541, 0.50041264295578, -0.5333449840545654, 0.3500198423862457, -0.8888645172119141, 1.5273364782333374, 1.055821418762207, 1.204283595085144, 0.16691255569458008, -0.17817910015583038, -0.26198241114616394, 2.293943166732788, -1.1918154954910278, 1.338067650794983, 0.2775207459926605, 0.6840716004371643, -2.1952946186065674, -1.7805116176605225, -0.3182563781738281, 1.2468340396881104, -0.4423127770423889, -0.676155686378479, -0.10396357625722885, 0.160980686545372, 0.0320984423160553, 0.31886112689971924, 1.8397190570831299], [0.6934478282928467, 1.090995192527771, -1.3515651226043701, -0.22010602056980133, -1.7260968685150146, -1.1421496868133545, 0.8210367560386658, -0.15160752832889557, 1.0244885683059692, -1.7208311557769775, 0.9488098621368408, -1.1587905883789062, 1.1654918193817139, 1.8338301181793213, 0.29979845881462097, -0.05268168821930885, 0.2956058979034424, 0.10152676701545715, 0.6455908417701721, 0.6860122084617615, -0.9197449088096619, -0.3999248743057251, -1.4957212209701538, 0.11260337382555008, 0.7996162176132202, 0.3163531720638275, 1.0956804752349854, -0.8822205066680908, -0.1156737357378006, 0.8993182182312012, -0.6394243836402893, -0.5274056196212769, 0.6473377346992493, -0.19977569580078125, 0.7897049784660339, 1.0759499073028564, 0.5355287194252014, 0.24325032532215118, -0.34057095646858215, 0.029386017471551895, -2.2559802532196045, 1.034054160118103, 1.4559326171875, -1.3909742832183838, -1.7030398845672607, -1.0547553300857544, 0.6573874950408936, 1.5496093034744263, -1.7468547821044922, 0.2775922119617462, -1.3526508808135986, 0.34177476167678833, 0.7076907753944397, -1.5287644863128662, 0.05152711644768715, -1.7581690549850464, 0.5216089487075806, 1.1884658336639404, -1.7163035869598389, 0.586729109287262, -0.7811490893363953, -1.0400488376617432, 0.8742719888687134, -1.5092098712921143, -1.22202467918396, -0.21485663950443268, -1.3708133697509766, -0.5094154477119446, -1.2526497840881348, -0.29042237997055054, 0.48892953991889954, 0.4979623556137085, 0.15099193155765533, -1.0165537595748901, 0.3431851267814636, 0.6158761978149414, 0.26061904430389404, -2.0628724098205566, 0.7459222078323364, -1.9268442392349243, 0.0638095811009407, 0.7633430361747742, -0.1118764877319336, 0.06878482550382614, 0.6874029636383057, 0.20844268798828125, 0.12314057350158691, 0.7405550479888916, 0.750372052192688, 0.34282565116882324, 0.8799424171447754, -1.4805546998977661, 0.6098882555961609, 0.39467817544937134, 2.1407713890075684, 0.947003960609436, 0.36496496200561523, -0.291861891746521, 1.3724595308303833, -0.4716796576976776, 0.6099894046783447, -1.1688475608825684, -1.3421107530593872, -0.9176647067070007, 0.9010252356529236, -1.0318946838378906, 1.806107759475708, -1.290700078010559, 0.030277837067842484, -1.2125639915466309, -1.8223894834518433, -0.83628910779953, -2.7717573642730713, 1.0593369007110596, -1.2050156593322754, -0.7728058695793152, -1.4660216569900513, -0.7481356859207153, -0.055751651525497437, -1.750177025794983, 1.1439262628555298, 0.18757611513137817, 0.13440091907978058, 1.268250823020935, 1.4712491035461426, -0.4757695198059082, 0.5263124704360962, 0.5393264293670654, 0.9082134366035461, -0.10097702592611313, -0.5745063424110413, -0.20223163068294525, -0.005274360068142414, 0.7574214935302734, -0.5826184153556824, 0.9820151925086975, 1.1785578727722168, 1.101926565170288, -0.41531142592430115, -0.7398375272750854, -2.4766507148742676, 0.15292929112911224, -1.5622129440307617, 0.8731486201286316, -0.2061968296766281, -1.4486217498779297, -0.9989209771156311, -0.8178068399429321, -1.07590913772583, 0.9678515195846558, 0.7657564282417297, -0.7802250981330872, -0.11497876793146133, -0.45633840560913086, 0.2166779637336731, 0.03245103359222412, 0.40861931443214417, 0.3296784460544586, 1.5313384532928467, -0.7886613011360168, 0.7442718148231506, 1.7956596612930298, -0.5458420515060425, -0.07867952436208725, 0.6293249726295471, 0.006337390281260014, -0.10939357429742813, 1.7804723978042603, -1.1828153133392334, 1.1969398260116577, -0.6519852876663208, -0.5076110363006592, -0.25483784079551697, -1.230409860610962, 0.7830904722213745, 0.5049088597297668, 1.1058721542358398, -1.7183282375335693, 0.4677375257015228, 1.0704431533813477, 0.3004032075405121, 0.09693657606840134, -1.459492564201355, -1.771755337715149, -0.3752565085887909, -0.6845338940620422, 0.7639515995979309, -1.8811830282211304, -0.43545886874198914, 0.5824216604232788, 0.5803884267807007, 1.1747251749038696, 2.2296063899993896, 1.2236438989639282, 0.22354689240455627, 0.10885465890169144, -0.8721537590026855, 0.8262130618095398, 0.7856612205505371, 0.9009132981300354, -1.4146968126296997, 2.451897621154785, -0.2988015115261078, 0.007638493552803993, 1.2918988466262817, 1.2249170541763306, 0.33970576524734497, 0.7421813607215881, -0.1823512464761734, -1.0987752676010132, 1.0887259244918823, 0.5067527294158936, -0.14537201821804047, 1.4646562337875366, 0.3208729326725006, -0.7584762573242188, 0.9053065180778503, 0.11717090755701065, -0.7113844156265259, 0.522125780582428, -1.4393420219421387, -0.9931156039237976, 0.9388908743858337, -0.35052579641342163, -0.9196362495422363, -0.6837271451950073, -0.33432891964912415, -0.4499676525592804, 1.9510409832000732, -1.1261770725250244, 1.1746405363082886, 0.42275819182395935, -0.2444949597120285, -1.3055046796798706, 1.958648920059204, 0.7453873157501221, 0.8284882307052612, 0.02658529207110405, -1.3709332942962646, -1.3851522207260132, 0.8331725597381592, 0.41624167561531067, -1.466088891029358, -0.5625103712081909, 1.3346704244613647, 1.323304295539856, 1.1903563737869263, -0.012130063958466053, -1.0779032707214355, 1.0374454259872437, -0.6518636345863342, 1.1500537395477295, 1.0692558288574219, 0.6259669661521912, 0.16516250371932983, 0.6614658236503601, -1.293350100517273, 0.5600898861885071, -0.8068587779998779, -1.379133939743042, 0.0038307742215692997, 0.6680234670639038, -0.5759791731834412, -1.1126903295516968, 0.7801672220230103, 0.325528085231781, -1.0082263946533203, -0.21337635815143585, 1.1177235841751099, 1.2993069887161255, -0.148357555270195, 0.8960827589035034, -1.2950865030288696, 1.1237205266952515, -0.33648937940597534, -0.05637671425938606, -0.49493587017059326, 1.867678165435791, 0.406948983669281, -0.7923416495323181, 0.36985528469085693, -1.5614609718322754, 1.1640024185180664, -0.17536868155002594, -0.14623168110847473, 0.6812058687210083, 0.44310277700424194, -0.3589152693748474, -1.5018680095672607, 1.8340634107589722, -0.6435441374778748, -0.47811102867126465, -0.4936526119709015, -0.09739349037408829, 2.165325164794922, -0.15879075229167938, 0.6816381812095642, -0.4209233224391937, -1.728026270866394, 0.37807509303092957], [-0.32779237627983093, -0.483707457780838, -0.9159829616546631, -0.16834408044815063, 2.4018807411193848, 1.3709022998809814, -0.26488447189331055, 0.8238281607627869, -0.036760661751031876, -1.1511975526809692, 0.8242766261100769, -2.3745107650756836, 0.53071528673172, -0.446058064699173, 0.21479901671409607, 1.4774675369262695, -0.7819311022758484, 2.3648922443389893, 0.23398761451244354, 0.936919093132019, 1.07666015625, -1.2116212844848633, -0.6283289194107056, -0.33168262243270874, 0.5984352827072144, 1.0256516933441162, 1.4498286247253418, 0.41735172271728516, 1.8983378410339355, -1.6974760293960571, -0.4867457151412964, -0.506842851638794, 0.21677260100841522, 0.12340760976076126, 1.7913011312484741, -1.2140388488769531, 0.5657222270965576, -0.8377188444137573, 0.7209571599960327, -1.731569766998291, 0.1516360342502594, -1.1724367141723633, -1.0156300067901611, 1.5194027423858643, -0.8513580560684204, 2.3247225284576416, -0.7456914186477661, 0.0036321161314845085, 0.9368054270744324, -0.2398984581232071, -0.21271072328090668, -0.5026572942733765, -0.8739460110664368, -0.6847817897796631, 0.32874971628189087, 0.23466463387012482, -1.292891502380371, 1.2599354982376099, -0.7159798741340637, -0.8626799583435059, -0.16398993134498596, -1.754243016242981, 1.0980415344238281, 0.5408088564872742, -0.39850524067878723, 0.16857200860977173, 1.0876368284225464, 0.6056008338928223, -1.3973995447158813, -0.013161404058337212, 1.416938066482544, -1.393798589706421, 1.2486486434936523, -1.3205050230026245, 1.0049477815628052, 1.025142788887024, 0.053500305861234665, -0.05432958900928497, -0.3374568521976471, -0.008885197341442108, 0.3895591199398041, -1.3156567811965942, -0.22481152415275574, 0.8470418453216553, 0.7028454542160034, -0.7789466977119446, 0.15296857059001923, 0.5858432650566101, 0.05959879979491234, -0.29278314113616943, 0.5696679949760437, -1.747074007987976, 1.490026593208313, -0.19242426753044128, -0.031425584107637405, -0.47474557161331177, -0.43462979793548584, 0.4431820213794708, 1.2392536401748657, -0.35482966899871826, -2.0464072227478027, 0.5637409090995789, -0.5813233852386475, -0.13313764333724976, -0.5074290633201599, 1.347303867340088, 0.4231550395488739, 0.5463137030601501, -1.0731337070465088, 0.9456139206886292, 0.10176688432693481, -0.7282530665397644, -0.6444732546806335, 0.000848260591737926, -0.05180361121892929, -0.2142123281955719, 0.025785373523831367, -1.0166412591934204, -0.7058192491531372, 0.6571656465530396, -1.2017847299575806, 0.08836595714092255, 0.9021812081336975, 1.0362389087677002, -1.4004945755004883, -0.8254151344299316, 0.4710814654827118, 0.9305857419967651, 0.19984577596187592, -0.06223277747631073, 0.43244969844818115, -1.0098222494125366, 1.09683096408844, -0.042650092393159866, -0.38931378722190857, -1.2621891498565674, 0.688249409198761, 0.5315739512443542, -1.3895666599273682, -0.3685120940208435, 0.6022547483444214, -0.46192315220832825, 0.3045783042907715, 0.3883303701877594, -0.0155373839661479, 0.3619861602783203, -2.1695072650909424, 0.05522982031106949, -1.1243987083435059, -1.001318097114563, -1.1272313594818115, -0.8124816417694092, -0.911365270614624, 0.00926879234611988, 0.9430603981018066, 0.8866583704948425, -1.8046119213104248, -0.545908510684967, 0.019532794132828712, -1.8218955993652344, 1.2309954166412354, 0.9784566760063171, 0.18507279455661774, -0.7668899297714233, -0.18441124260425568, -0.15233677625656128, -0.9618004560470581, -0.962491512298584, 0.31733348965644836, 0.10844551026821136, 0.6283509135246277, -0.6177580952644348, 0.7260721325874329, -1.21851646900177, 0.3054504692554474, 0.25472041964530945, 0.9772690534591675, 1.0396686792373657, 0.08096487820148468, -0.0026939690578728914, 2.4012458324432373, 2.004411458969116, 0.07364913076162338, -2.718456745147705, 1.674057960510254, 0.6033291816711426, -0.049358028918504715, 0.7607626914978027, 0.14132601022720337, 1.3767811059951782, 0.7581650018692017, -0.3026421368122101, 1.858422875404358, -0.6270845532417297, 1.1415534019470215, 0.8716986179351807, -0.0583166740834713, -1.556694746017456, -0.6474646329879761, -0.7747563719749451, -1.0851943492889404, 0.8416352868080139, -2.5886688232421875, -1.7222809791564941, 0.6937651634216309, -0.016779974102973938, 1.857692003250122, -0.9498978853225708, 1.627583384513855, -1.3671249151229858, -2.339855670928955, 0.05933074653148651, 2.013007164001465, 0.908075749874115, -0.18417072296142578, 2.4545867443084717, 0.0067546917125582695, 0.9116527438163757, -0.004078807309269905, 1.2976380586624146, -1.3986819982528687, 0.21774394810199738, -2.4333932399749756, -0.5488309860229492, 0.27436134219169617, -1.3118069171905518, -0.6570638418197632, 0.2239399552345276, -0.24042025208473206, -0.07613848149776459, 0.9296702742576599, 0.19287890195846558, -0.13306894898414612, 2.028683662414551, 0.6501161456108093, -0.4305266737937927, 1.8621083498001099, -0.853463351726532, -1.7034988403320312, -1.245123028755188, -0.569248378276825, -0.5166257619857788, 0.2836078703403473, -1.6146302223205566, 0.9036696553230286, -0.0003663297393359244, -0.6449379920959473, -0.6797109842300415, -1.3865690231323242, 1.356239676475525, 0.48545774817466736, 0.438386470079422, -0.23580585420131683, -0.14466899633407593, 1.8915835618972778, 1.6334013938903809, -0.8373045325279236, 1.5403282642364502, -0.8919476270675659, 1.356757402420044, -0.27102264761924744, 1.065946340560913, -1.2425098419189453, 0.706951379776001, 2.706824779510498, -0.6694754362106323, 1.098552942276001, 1.0246899127960205, 0.44744598865509033, 0.012894639745354652, 0.5687102675437927, -0.9000802040100098, -1.426621437072754, -0.2177436798810959, 0.8151736855506897, -0.5071545839309692, -0.7217847108840942, 0.43688300251960754, 0.038067515939474106, -0.2891877591609955, 2.686436176300049, 0.9110216498374939, 0.021824324503540993, 0.11822901666164398, -1.616909384727478, -0.649628758430481, 1.6101995706558228, 1.2639328241348267, -0.016377434134483337, 0.5664588809013367, 2.6107430458068848, -1.654557466506958, 0.03502622991800308, -1.302170753479004, 0.21968647837638855, -0.933445155620575, -0.5948298573493958, -0.7029131650924683, -0.8643575310707092, 1.2194442749023438], [-0.4606349766254425, -1.067179799079895, 0.19110578298568726, -1.0311167240142822, 0.4215719699859619, -0.7419193387031555, -0.12140847742557526, 0.5533282160758972, -0.71055006980896, -0.47669729590415955, -0.046588484197854996, 0.6415408253669739, -0.13355839252471924, -0.9410960674285889, 0.0581645630300045, -0.7970799803733826, -2.9701406955718994, 1.394289255142212, 0.9156304597854614, -0.1484275460243225, 0.6256049275398254, 1.1574821472167969, 1.1301913261413574, -1.5427501201629639, -0.9412893652915955, 0.7657110095024109, 0.45036250352859497, 0.5564951300621033, -0.260168194770813, 0.0937705710530281, 1.4065163135528564, -0.6649980545043945, -0.7285573482513428, 0.2380198985338211, 0.17581918835639954, -1.6936159133911133, 1.1106373071670532, 2.1522343158721924, -0.028822917491197586, 0.35828372836112976, -1.3085631132125854, 1.0512523651123047, 0.5641670227050781, -1.1334806680679321, 0.5905518531799316, 0.47625479102134705, -1.2731647491455078, -0.22933349013328552, 0.3428454101085663, 0.35981646180152893, 0.8981400728225708, -0.9898239970207214, -0.7660013437271118, -0.8075809478759766, 0.7067681550979614, -0.3248782455921173, 1.0059396028518677, 0.9470755457878113, 0.6046246290206909, 1.0614920854568481, 0.4333869516849518, -0.425590842962265, -0.7923080325126648, 0.7464881539344788, 1.3966197967529297, -1.8644959926605225, -0.7815767526626587, -0.7085505723953247, -0.20182375609874725, -0.6431450247764587, -1.1725819110870361, -0.4830295741558075, 1.0584536790847778, 0.10875890403985977, -1.4039463996887207, -0.9040013551712036, -0.257257878780365, -0.501069188117981, 1.7620958089828491, -0.0058481222949922085, 1.0805227756500244, -0.031351592391729355, -0.43442752957344055, 0.6547757983207703, 0.04057328402996063, 0.005062229000031948, -0.24575629830360413, -0.7418663501739502, 0.7525857090950012, 0.5278199911117554, -0.5601892471313477, -0.37137407064437866, 1.752478003501892, 1.3567407131195068, -0.670434296131134, 0.3815854489803314, -0.6360869407653809, 0.268907368183136, 0.3659977912902832, 0.27205875515937805, -0.4581301212310791, 0.1737160086631775, -1.426108479499817, -0.33820483088493347, 0.3615892231464386, -0.8259152173995972, 1.039794921875, 0.8269540071487427, -0.5516261458396912, 0.602215588092804, 0.07916024327278137, -0.5669898986816406, 0.5148271322250366, -0.22414347529411316, -0.24371932446956635, -1.4068560600280762, -0.7284613847732544, -0.5284856557846069, 1.1292264461517334, 1.5509281158447266, -0.6310692429542542, -1.1690317392349243, -1.5980925559997559, 0.8730402588844299, -0.19777002930641174, 0.4717582166194916, 1.5859161615371704, 1.7739168405532837, -0.2726869583129883, -0.5841122269630432, 1.098173975944519, -0.058620158582925797, 0.4212134778499603, -0.049148909747600555, 0.12071766704320908, 0.5678046345710754, -0.7295017838478088, -0.08103522658348083, 0.34303581714630127, -1.2341231107711792, -0.5865055918693542, -0.5187705755233765, 1.7003015279769897, 1.2993139028549194, 0.002471149666234851, 0.8562652468681335, 0.5044726729393005, -0.3331839442253113, 0.3082219362258911, 0.18488401174545288, -0.5078082084655762, -1.5782960653305054, 1.1387547254562378, -1.1130216121673584, 1.9734772443771362, -1.4624137878417969, 0.7554197907447815, 0.4704326093196869, 1.5523282289505005, 0.7692286968231201, 0.6789739727973938, 0.031625788658857346, -1.309517741203308, 1.1809109449386597, 0.43081212043762207, -0.5333152413368225, 0.7730894684791565, -0.6310949921607971, 0.03851472958922386, 1.3366634845733643, -0.008474251255393028, -1.5418137311935425, -1.0928542613983154, -0.27306950092315674, -0.7759804725646973, 0.4169541001319885, 0.2897566258907318, -0.8858906030654907, -1.3539992570877075, 1.0384716987609863, -0.6462870240211487, 0.871724545955658, 0.39714837074279785, -0.4011097252368927, 0.4009827673435211, -0.3574599027633667, -0.3143153786659241, 0.03276060149073601, 1.1069395542144775, 0.7719018459320068, 0.5593129992485046, 2.6197350025177, 1.140803575515747, 0.5008591413497925, 0.6364353895187378, 0.28287020325660706, -0.2794986367225647, 1.4745243787765503, -0.10781217366456985, -1.7755593061447144, -0.6112074255943298, 0.5988603234291077, 2.9523186683654785, 0.6328005790710449, 0.33008408546447754, -0.8598102331161499, 0.021039318293333054, 1.0238194465637207, 0.28179147839546204, -0.29115256667137146, -0.13716761767864227, -1.1260894536972046, 1.4657015800476074, -0.15299645066261292, 0.11565375328063965, -0.14026156067848206, 0.7730109691619873, 0.27920418977737427, -0.7704079151153564, -1.3058689832687378, 0.2672114670276642, -1.1121004819869995, 0.3651603162288666, -0.48932093381881714, 1.8095386028289795, -1.334232211112976, 0.882495641708374, -0.7635034918785095, 0.17759016156196594, 1.254941463470459, -1.1383920907974243, -0.8301878571510315, -2.273550271987915, -2.1964781284332275, 1.473450779914856, -1.0346081256866455, -1.1456419229507446, 0.307282954454422, -0.8065626621246338, 0.18205960094928741, 1.5865418910980225, 1.0913567543029785, 2.5588738918304443, 0.4699545204639435, -0.25421810150146484, 0.7599771022796631, 0.558806836605072, -1.800519585609436, -0.9899024963378906, 0.2581535875797272, -0.4692825973033905, 1.4752798080444336, -0.2456749826669693, 0.8472134470939636, 0.4472593665122986, -0.9865283966064453, 0.7460103034973145, -0.3721940815448761, -1.462001085281372, 0.09282734990119934, -0.020973093807697296, 1.0115902423858643, -0.41847655177116394, -0.49609339237213135, -1.6408028602600098, 0.8691883683204651, -0.2914450764656067, 1.4405685663223267, -3.053427219390869, -0.6581843495368958, -0.27556008100509644, -0.549985408782959, 0.789936900138855, 0.7446766495704651, -1.3228633403778076, -0.5269610285758972, 1.1134066581726074, 0.2633819580078125, 1.6102280616760254, 1.285857081413269, -0.20762331783771515, 0.045281440019607544, 0.3244951367378235, -0.5804586410522461, 0.5056878328323364, 0.1426377147436142, -0.31975072622299194, -0.9632534980773926, 0.5152408480644226, 0.4047091007232666, -2.6601877212524414, -0.2584768235683441, 0.6104576587677002, -0.024779871106147766, 0.19021183252334595, 0.31904321908950806, 1.1695442199707031, 0.25229814648628235, 1.6923606395721436, -0.09823095053434372], [2.41861629486084, -1.368155837059021, 0.18152377009391785, -1.22415030002594, -0.4618220329284668, -1.5528528690338135, -1.0692734718322754, -0.8013982176780701, 0.5431031584739685, 0.10588525980710983, -0.2989216148853302, 0.749448835849762, 0.5420876145362854, 0.32366183400154114, 0.4397141635417938, -0.5119082927703857, -0.9324418902397156, -0.713299572467804, -0.6665864586830139, -0.9563136696815491, -0.31439751386642456, 1.3162082433700562, 0.24595005810260773, 0.11539953947067261, 0.6422348022460938, -1.549241304397583, -0.7118923664093018, 0.21924743056297302, -0.05780305713415146, 0.26837438344955444, 1.454689860343933, 0.6415647864341736, -0.07922731339931488, 0.4573812782764435, -0.1050693467259407, 1.0991710424423218, -1.3096204996109009, -1.3004664182662964, 1.0726953744888306, -2.2792301177978516, 0.3374503254890442, -0.4159201383590698, 1.817874550819397, -1.0559810400009155, 0.7267479300498962, -0.13255159556865692, -1.8999738693237305, -0.6975345611572266, -0.5906798243522644, -0.20196977257728577, -0.008885720744729042, 1.1816632747650146, 0.16522300243377686, 0.1376955211162567, 0.40711164474487305, -0.2803176939487457, 3.2107949256896973, 0.6393720507621765, -0.05632125586271286, 0.0806141048669815, -0.5798670649528503, 0.02705416828393936, -2.103248357772827, -0.739954948425293, 0.15390290319919586, 1.2717336416244507, 0.07093051820993423, -0.5202938318252563, 1.1514946222305298, 1.4188237190246582, -0.9807092547416687, 1.5306495428085327, 1.5214482545852661, 0.6142199039459229, -0.35253435373306274, 0.07010595500469208, -1.4583803415298462, 0.9040980935096741, 1.3137097358703613, -2.106539487838745, -0.3625037372112274, 0.9691060185432434, 1.1516718864440918, 0.7438061237335205, 0.7923405766487122, 1.456600308418274, 0.5519587993621826, 1.0231962203979492, -0.4755968749523163, 0.6841953992843628, -1.1704760789871216, 1.3891035318374634, -0.956066906452179, 0.14556297659873962, -1.1062021255493164, -1.3871734142303467, 1.1545743942260742, 1.712911605834961, 0.9668784737586975, 0.46368253231048584, -1.8161243200302124, 0.8756281137466431, -0.6166937351226807, -0.7938659191131592, -0.5212467908859253, -0.1935110092163086, -0.29660043120384216, -0.7901844382286072, -1.1827067136764526, 0.3836604356765747, -0.2173856645822525, -0.4555664360523224, -1.1694058179855347, 0.6581623554229736, -0.5141440629959106, 0.12110933661460876, -0.3029364049434662, -0.2902437746524811, 0.8315865993499756, -0.6712300181388855, -1.1541486978530884, 0.5409283638000488, 0.10985304415225983, 0.1945948600769043, 0.3407266139984131, 0.5203865766525269, 1.941824197769165, 0.010324367322027683, -1.60745108127594, -1.19320809841156, -0.9329147934913635, -1.308932900428772, -1.0295132398605347, 0.5734597444534302, -0.4848860204219818, -0.41917991638183594, 0.1334831863641739, 0.159549742937088, 0.16284441947937012, -0.669969916343689, -0.3189406394958496, -0.40724584460258484, 2.5765349864959717, 1.0101872682571411, 0.07680600881576538, -0.3836902379989624, 2.9009408950805664, 0.36704978346824646, 0.2529537081718445, 0.9530632495880127, 1.0790024995803833, 1.3540821075439453, 1.475311517715454, 1.8458017110824585, 1.3382847309112549, -0.3622826039791107, 0.3233087658882141, -1.3550875186920166, -0.3007616698741913, -1.3795006275177002, 0.8779040575027466, 0.37418267130851746, 0.08810651302337646, 1.320436716079712, 0.5777202248573303, 3.6165716648101807, 0.2093421220779419, 0.5344312191009521, -0.34663262963294983, -1.0095467567443848, -0.246419757604599, -1.4706406593322754, -0.11859466880559921, 1.425012230873108, 0.06777432560920715, 1.3375005722045898, 1.8203223943710327, 0.27440401911735535, -0.6211496591567993, 0.7001357078552246, 0.5153841972351074, -1.8368256092071533, 2.2063305377960205, 0.9788559079170227, -0.38975778222084045, 0.5070961713790894, 0.25235942006111145, 1.1525154113769531, -1.2776271104812622, -0.2406240552663803, -2.7834432125091553, 0.29987066984176636, -1.619057536125183, -0.5974709987640381, 0.5682228207588196, -1.832134485244751, 0.45687010884284973, 0.1485661268234253, 0.6988354921340942, 0.3124914765357971, -1.0137827396392822, -0.11093586683273315, 0.7908252477645874, -0.5484852194786072, -0.7368237376213074, 1.250939130783081, 1.6078242063522339, 2.137483596801758, 0.6186230182647705, -0.5526345372200012, -0.7849075198173523, -0.883998453617096, 0.423585444688797, -0.48030707240104675, 0.9254111051559448, -0.14543499052524567, 0.2479850798845291, -1.723288655281067, 0.1479233205318451, -2.052307605743408, 1.8341587781906128, 1.072486400604248, -1.5415737628936768, 1.560828685760498, 0.03751703351736069, -1.3490824699401855, -0.12694953382015228, -1.4382516145706177, -0.28632378578186035, -0.31354475021362305, -0.2603098154067993, -0.5912561416625977, 1.1285803318023682, -0.3913191258907318, -1.6789568662643433, -0.29638224840164185, -0.6436643600463867, -1.9233880043029785, 0.13153308629989624, -2.1651253700256348, -0.09215615689754486, -0.14025099575519562, 2.013089656829834, 0.6218313574790955, 1.180451512336731, -0.6535186767578125, -0.2773883640766144, -1.2741847038269043, 0.6179316639900208, -0.8081080913543701, 0.20560817420482635, 0.4455590546131134, -0.12616418302059174, -0.8300012350082397, -1.9737433195114136, 0.8813586831092834, 1.2549525499343872, -0.08864133059978485, 1.8057609796524048, 0.8545173406600952, -1.1630895137786865, 0.6762029528617859, 1.3799824714660645, 0.5873598456382751, 0.534135103225708, -0.7461788654327393, 0.17325136065483093, 0.32008713483810425, 0.06285350024700165, -0.8469100594520569, -0.9388238787651062, -1.5256969928741455, 0.67796790599823, 1.5842108726501465, 0.25576531887054443, 0.5194119811058044, 1.271710991859436, -0.23053386807441711, 1.4387489557266235, -1.0198816061019897, 3.5564253330230713, 0.3376195430755615, -1.4019320011138916, 0.943917989730835, 0.9679545164108276, -0.9765459299087524, 1.1918871402740479, -0.5889379382133484, 0.24567528069019318, 0.7373782992362976, 0.8316723704338074, 0.3832402527332306, -0.43639206886291504, 0.32430291175842285, -0.13297735154628754, -1.2415491342544556, 0.1098133847117424, 0.6302959322929382, 0.10152275860309601, -0.39911872148513794], [1.0574595928192139, -0.28387537598609924, -0.5179241895675659, 0.44815945625305176, 1.1529730558395386, -0.7186373472213745, -1.365340232849121, 1.697407603263855, -0.35060587525367737, -2.301368236541748, 0.7794153094291687, 1.067613124847412, 1.2628116607666016, -0.35622185468673706, 0.045203160494565964, -0.0458146370947361, 1.3392653465270996, 1.3503769636154175, 0.054769162088632584, 0.12212907522916794, 1.200756311416626, 1.8203990459442139, 1.308115005493164, -1.3056048154830933, 0.22337616980075836, 0.6425392031669617, 1.4733295440673828, -0.41738954186439514, 1.3591711521148682, -0.28108713030815125, -2.3410377502441406, -1.0805795192718506, -0.39682820439338684, 0.05322325602173805, -0.4238582253456116, -1.3907486200332642, 0.8305604457855225, -0.4393990635871887, 0.9223718047142029, 0.6546695828437805, -0.04947417229413986, -1.288025975227356, -0.9955905079841614, 0.475469172000885, 1.00384521484375, -0.12247384339570999, -1.7087782621383667, -1.5652847290039062, -2.733621120452881, -0.9053704142570496, 1.2225465774536133, -0.9570699334144592, -1.5377012491226196, 0.33645492792129517, 3.6101267337799072, 1.031424641609192, -1.1710727214813232, 0.18004117906093597, 0.37763121724128723, 0.4185800850391388, -0.49437394738197327, 1.484316349029541, -1.188429832458496, 0.7707185745239258, -1.7264255285263062, 1.10024094581604, -0.4925198554992676, 0.3661957085132599, -0.159764364361763, -0.04235910624265671, -0.38210445642471313, -0.7484680414199829, 0.2896859049797058, -0.5750173926353455, 0.04599348455667496, 1.1327824592590332, 0.7400806546211243, 1.1735374927520752, -1.4691787958145142, -0.8863084316253662, -0.6227192878723145, -0.02719978615641594, -1.8990765810012817, -0.22091367840766907, 1.3049391508102417, -0.8450630307197571, -2.087648391723633, -1.1755386590957642, -0.6833855509757996, 0.5922675728797913, -0.3483254909515381, -0.8315457105636597, -0.9482857584953308, -0.9233075976371765, -1.0749956369400024, 0.15585783123970032, 1.8469886779785156, -2.410229444503784, -0.24196135997772217, -0.8291823863983154, -0.8384175896644592, -0.44349247217178345, -0.6657405495643616, -0.7731376886367798, -0.8388552665710449, 1.5267146825790405, 1.5319209098815918, -0.5318452715873718, 0.03205202519893646, 0.10779044777154922, -2.0625603199005127, -0.7888603806495667, 2.637615919113159, 0.6705136895179749, -0.2383173406124115, 1.0005834102630615, -0.5758112072944641, 2.7051639556884766, 0.029049821197986603, 0.616386353969574, -2.320053815841675, 1.4737427234649658, -1.3424409627914429, 0.1827363669872284, 0.09433409571647644, 0.1454859972000122, 1.317286491394043, -1.7925775051116943, -0.08497506380081177, -0.9159955978393555, 0.5555163621902466, -1.6031838655471802, 1.0571693181991577, -0.4541434347629547, 0.44653987884521484, 0.6768098473548889, -0.32866719365119934, -0.33425554633140564, -1.317509651184082, -1.2323497533798218, -0.7983784079551697, 0.18907050788402557, -0.27083927392959595, 0.21893101930618286, 1.1113988161087036, -0.853158712387085, 0.8830159902572632, 0.37192317843437195, 0.16996343433856964, 0.436086505651474, -1.6531615257263184, 0.9200866222381592, -1.221266269683838, -0.06800446659326553, -0.9700146317481995, 0.6470662355422974, -0.7381625771522522, -0.2160843014717102, -0.4546492397785187, 0.95164555311203, -1.6240381002426147, -0.2949318289756775, 0.72950279712677, 0.13254700601100922, -0.20282164216041565, -0.5450573563575745, 1.661362648010254, -0.37940695881843567, -0.04709475487470627, -2.0314111709594727, -1.3722131252288818, -0.7214447855949402, -1.1175087690353394, 1.096610426902771, 0.029950952157378197, -0.580425500869751, 1.2801066637039185, -1.4433221817016602, 1.4359025955200195, -0.24025915563106537, 1.9457124471664429, -0.7835709452629089, 0.26966962218284607, -0.2784065306186676, -0.6575357913970947, 0.5447043776512146, -0.36102089285850525, -1.6005288362503052, 0.4273805320262909, -0.28167787194252014, -0.28002050518989563, 1.939658284187317, 0.787749707698822, 1.3383466005325317, -0.780695378780365, 1.0648605823516846, 1.1906192302703857, 0.505876362323761, 0.8618385195732117, -0.34912359714508057, -0.4125917851924896, 0.08441568166017532, 0.7736784219741821, -0.8300111889839172, 0.04807666316628456, -0.6469013094902039, 1.021284818649292, -2.4327845573425293, -0.9940763711929321, -0.9221237301826477, -1.1290538311004639, -1.0322885513305664, 1.8515691757202148, 0.09584501385688782, 1.0226404666900635, 0.26702016592025757, -0.22246801853179932, 0.11252418160438538, -0.2258053869009018, 0.7387191653251648, -2.4887471199035645, -1.4534735679626465, -0.34121519327163696, 0.30904850363731384, -1.749029278755188, -1.1059494018554688, 1.4577192068099976, -0.6038894653320312, 0.5020312070846558, -0.915809154510498, -0.3419535160064697, 1.1445528268814087, -0.35405242443084717, -1.162927508354187, -0.8234214782714844, 1.5505225658416748, -0.11531469970941544, 1.7436201572418213, 0.48273253440856934, -0.4850676953792572, -0.675647497177124, 0.33245500922203064, 0.9175998568534851, -0.23680278658866882, -1.7453268766403198, 0.18722248077392578, 0.6494253873825073, 2.15035080909729, 0.9381536245346069, -0.8199633359909058, 0.4039097726345062, 0.24532312154769897, -0.25070858001708984, -0.07477012276649475, 0.5310323238372803, 0.2036447823047638, -1.5030100345611572, 1.596123218536377, -1.3427402973175049, 0.09306754916906357, 0.04478473216295242, 0.8202309012413025, -0.06524797528982162, 1.377982497215271, 0.20340783894062042, 0.255331814289093, 0.5048788785934448, -1.298365831375122, -0.7451727986335754, 0.09860940277576447, 1.659119963645935, 2.2040631771087646, 1.2943772077560425, -0.4637715220451355, -0.6346287727355957, -1.2230098247528076, 0.14654353260993958, 1.0523310899734497, 0.8629416823387146, -0.22898511588573456, 0.38643473386764526, -0.7911387085914612, -0.42841383814811707, 0.5555994510650635, -0.9051070213317871, -1.7646604776382446, -1.1981159448623657, -0.016482815146446228, 1.1677619218826294, -0.547427773475647, -1.34860098361969, -1.6874055862426758, 0.47577160596847534, -1.3714518547058105, -0.2435721606016159, 1.1972758769989014, -0.5148721933364868, 0.6114362478256226, -0.15891681611537933, -1.570337176322937], [-1.5082695484161377, -0.7006202936172485, -0.38818323612213135, 1.220708966255188, 0.7725735902786255, 0.08007165789604187, 1.278967022895813, 0.46592825651168823, -0.12753288447856903, -0.235850527882576, 0.2960064113140106, -0.4112463593482971, 0.4035109281539917, 0.8882724642753601, 1.5971759557724, -1.1153428554534912, -1.5320993661880493, 0.6970123648643494, 0.7550463080406189, -0.5005429983139038, -0.18056249618530273, 0.8746885061264038, 1.5674186944961548, -0.16912494599819183, 0.5385704040527344, -0.46583548188209534, 1.0180248022079468, 0.2996029853820801, 1.1512442827224731, -1.5142498016357422, 0.14431139826774597, 0.09422540664672852, 1.2975385189056396, -1.068738341331482, -1.5131525993347168, -0.21358445286750793, 0.11860454827547073, -1.425994634628296, -0.13144104182720184, -0.7398325800895691, -0.9892991781234741, 1.2322578430175781, 3.083292007446289, 0.606282651424408, 0.5289758443832397, -0.012257849797606468, 0.8595882654190063, -0.4024507999420166, -0.5047242045402527, -1.989097237586975, -0.4487781822681427, -1.3250999450683594, -0.719190239906311, 1.580712914466858, -0.22604145109653473, -0.9902529120445251, 0.09507974237203598, -0.14077246189117432, 0.8867914080619812, -2.2776060104370117, -0.9884869456291199, 0.34993138909339905, -0.06690218299627304, 0.29156383872032166, 1.299022912979126, -0.5297566652297974, -0.7298969030380249, -1.2006639242172241, 0.8912189602851868, 0.2890545129776001, 0.7492824196815491, 0.5292794108390808, 1.2752108573913574, 0.46238797903060913, 0.7416614890098572, 0.47650307416915894, 0.47091057896614075, 0.9798206090927124, -0.5939907431602478, -1.0806341171264648, -0.33886921405792236, -0.3029623329639435, 1.7072560787200928, -0.14131343364715576, 0.38750725984573364, -0.09826827049255371, -0.6249281764030457, -0.011508950032293797, 0.4221040904521942, -1.4816765785217285, -0.09931183606386185, 0.6994583606719971, 1.0388201475143433, -0.5972734093666077, 1.4873888492584229, -0.28951433300971985, -0.30832600593566895, -0.37701645493507385, -1.6708801984786987, -0.8041086196899414, -0.8721064925193787, -2.0027248859405518, 2.5220787525177, -0.4292157292366028, 0.9352338314056396, 0.6443197131156921, -1.262263298034668, -0.803261399269104, -0.4886122941970825, -0.937347948551178, 0.43078091740608215, -1.6065229177474976, -0.7898463010787964, 0.24075964093208313, -0.5888239741325378, -0.3186887204647064, -0.6839656233787537, -0.2846534550189972, 1.6793365478515625, 0.5725292563438416, 0.09507500380277634, 0.704619824886322, 0.6351848244667053, -0.05172554403543472, -0.02328910492360592, 0.014549342915415764, -0.1266833394765854, 0.46474403142929077, 0.3498941659927368, -1.2610325813293457, 1.0203911066055298, 0.9213895201683044, 1.2154027223587036, -0.4107166528701782, 1.231872797012329, -0.3061912953853607, -0.13471157848834991, 1.823703646659851, -0.2760809659957886, 1.565692663192749, -0.5491048097610474, -0.09164336323738098, 0.7960597276687622, -0.38543760776519775, -0.7444559931755066, -2.721264123916626, -0.49950453639030457, 0.5706239342689514, -0.3921245038509369, -0.14866912364959717, 0.5771746635437012, -1.0823557376861572, -0.9739378690719604, -1.2686351537704468, 1.179084300994873, -1.0001413822174072, 0.39839935302734375, 1.1559079885482788, -1.101098895072937, -1.9988675117492676, 1.7165817022323608, -0.1090923473238945, 0.313942015171051, -1.4649667739868164, -0.8263054490089417, -0.26020681858062744, 0.16050858795642853, 0.9866735935211182, -0.7608342170715332, 0.6595982909202576, 0.31384551525115967, 0.6071833372116089, -0.9765514135360718, -0.12956927716732025, -1.5017943382263184, -1.0068367719650269, -0.9582496881484985, -1.677168369293213, -0.09271688759326935, -1.672658920288086, -2.3170814514160156, -0.29698026180267334, -0.3401644825935364, 0.3476821780204773, 1.074357271194458, 0.3961769938468933, 1.0634866952896118, -0.5266312956809998, 0.6483886241912842, 0.24626687169075012, -0.027218615636229515, 0.9181941151618958, 1.1719276905059814, -1.4181195497512817, 0.6867707967758179, 0.5589910745620728, -0.15110979974269867, 0.8746019005775452, 0.2707524001598358, -0.09332673251628876, 0.5421625971794128, -0.7325890064239502, 1.2425509691238403, 0.6690914630889893, -0.5782509446144104, -0.5813860297203064, 0.3177344501018524, 0.1114019975066185, 1.2672063112258911, -0.24798010289669037, 0.014881518669426441, 2.617941379547119, 0.30173972249031067, 0.871676504611969, -0.7203956842422485, -1.0089908838272095, 0.4530096650123596, 0.8034444451332092, -0.1973952203989029, -1.4992526769638062, 0.8382502198219299, 2.032372236251831, -1.1214282512664795, 0.5517507791519165, 1.278079867362976, 0.1191849336028099, 0.5616735219955444, 0.9182847142219543, 0.029762111604213715, 1.1863983869552612, -2.1475284099578857, -0.5114233493804932, 0.6452964544296265, -0.7120106220245361, -0.5921059250831604, -1.2080320119857788, 0.17971321940422058, 1.1000159978866577, 1.0035430192947388, -0.6604042649269104, -0.05499876290559769, -1.7583932876586914, -0.601757824420929, -0.4622552990913391, -0.13857580721378326, -1.9720429182052612, 0.36716726422309875, 0.7834581732749939, -0.4548453390598297, 2.016770839691162, 0.5978285074234009, 0.5977872610092163, 0.5909485816955566, 0.05289014056324959, 0.5803239941596985, -0.7489051222801208, -0.5117830038070679, 0.9047197699546814, 0.2889069616794586, 1.1302837133407593, -1.171461820602417, -0.17183806002140045, 1.852859616279602, 1.1687122583389282, 0.35569480061531067, 0.004597047809511423, 0.7206248641014099, 1.0953646898269653, 0.37624025344848633, -0.11092986166477203, 1.6641725301742554, 0.4453088939189911, -0.4003458023071289, -2.0536673069000244, 1.6883903741836548, 0.5700917840003967, 0.18851624429225922, 0.0831647589802742, -0.6669967174530029, -1.9605133533477783, 0.5636641383171082, 0.2591922879219055, 0.7463682889938354, -0.23001866042613983, -0.8915809392929077, 1.4146909713745117, 0.19548527896404266, -2.6611151695251465, 0.2568514049053192, 1.5089805126190186, 0.2892853915691376, -0.11944638192653656, 0.6466930508613586, -0.5167539119720459, 0.08595170080661774, 1.1206003427505493, -1.1238112449645996, -0.13835978507995605, -0.7203828692436218, 1.0636274814605713], [0.05604146048426628, -1.187772274017334, -1.6314054727554321, -0.5127107501029968, -1.2931846380233765, -0.8389806151390076, -1.06630539894104, -2.3969104290008545, -1.4874171018600464, 1.0609787702560425, -0.5055049061775208, -0.9881901741027832, 0.6232560873031616, -1.1668294668197632, -0.478676974773407, -0.9073220491409302, -0.41202813386917114, -0.15494851768016815, -1.2060905694961548, 0.9351584315299988, -1.8801711797714233, 2.1011853218078613, 0.9359009861946106, -0.12032804638147354, 0.8632674217224121, -0.36195871233940125, -1.1650240421295166, 0.22579021751880646, 0.052058737725019455, -1.2531596422195435, 1.0013686418533325, -0.06515385955572128, -1.9220997095108032, 0.20607495307922363, 0.2726685702800751, -0.8851108551025391, 0.2221192568540573, -1.348083734512329, 0.9525462985038757, -0.7943941354751587, 0.9022116661071777, 1.7827684879302979, -0.32275059819221497, 0.9294936060905457, -1.1252840757369995, 0.34610438346862793, 0.5987144112586975, 0.05991222336888313, 1.0380175113677979, 1.6076815128326416, 0.5268983840942383, -0.6954678893089294, -0.8753226399421692, -0.12228907644748688, -0.7692745923995972, 1.3076540231704712, 0.15914399921894073, 0.26970934867858887, -1.1592414379119873, -0.23526307940483093, 0.698143720626831, 0.24815970659255981, 0.5883782505989075, 0.2614109516143799, 0.7156517505645752, -2.6730082035064697, -0.8069222569465637, -1.2358499765396118, -0.32860442996025085, 0.3791363835334778, 0.3648735582828522, 0.16445960104465485, 0.9301554560661316, 0.9830724596977234, 0.5190930366516113, 0.4671664535999298, 0.08609253913164139, 1.4362016916275024, 0.993918240070343, 0.21467474102973938, 1.1160271167755127, -1.4654227495193481, 1.14286470413208, 0.3124774396419525, -2.0787553787231445, 0.23583266139030457, 0.6567054986953735, -0.44529810547828674, -0.5244589447975159, -1.0951465368270874, 0.8560250997543335, -0.2831650972366333, 0.6658269762992859, 0.37121129035949707, -1.8428531885147095, 1.3724185228347778, 0.8506909012794495, -0.7790073752403259, 0.971972644329071, 1.3380266427993774, -2.021852493286133, 1.6015172004699707, 0.9681057333946228, -1.0218180418014526, 1.4117462635040283, 1.2556663751602173, 1.91703200340271, -0.09822017699480057, 0.917878270149231, -0.3410249650478363, -1.121156930923462, 2.467655897140503, -1.5524375438690186, 0.0665150135755539, 1.6156673431396484, 0.9268676042556763, -0.48551684617996216, -0.03064279817044735, -0.18384304642677307, -0.0701015293598175, 0.10721825808286667, 0.3456957936286926, 0.6712561845779419, -0.1713356077671051, -0.31701216101646423, -0.05413764715194702, 0.6656957864761353, -0.45101484656333923, 1.8838356733322144, -2.039763927459717, -1.2005443572998047, 0.824704110622406, -1.3684399127960205, 0.2634546756744385, -1.2656594514846802, 0.9960262179374695, -0.04747201129794121, 0.07304484397172928, -0.7671025991439819, -0.8591301441192627, -0.6339766383171082, 1.3460485935211182, 0.16988995671272278, 0.5126466155052185, 0.004793424159288406, -1.9738889932632446, -1.2278485298156738, -1.0137126445770264, 1.75762939453125, -2.364506721496582, -0.09214258939027786, -1.4650912284851074, 0.574193000793457, -0.44172996282577515, -0.008499270305037498, -0.26351073384284973, 0.2132578045129776, -0.44971394538879395, -1.2866926193237305, 0.34306544065475464, -0.6327815651893616, 0.13880185782909393, 0.25649145245552063, 0.5740855932235718, 0.6817227005958557, -0.5863807797431946, -0.02955024130642414, -0.3728528618812561, -0.09339912980794907, -1.6265873908996582, 0.5583354830741882, -1.7436048984527588, 0.3515792787075043, 1.3872002363204956, -0.9859840869903564, 0.9364287257194519, 0.1369866132736206, -1.2130764722824097, -0.14327527582645416, 2.306077718734741, -1.7280203104019165, -0.4567853808403015, -0.6740901470184326, 1.3231489658355713, -0.9040349125862122, 0.3173510432243347, 0.12073897570371628, -1.177199125289917, 0.7647338509559631, 0.37570324540138245, 1.841504693031311, -1.2223609685897827, -1.874605655670166, 0.3687850832939148, -1.1181896924972534, -0.03618500009179115, 0.9165305495262146, -0.23110048472881317, -2.0787971019744873, -0.3803652822971344, -1.5773316621780396, 0.7119103074073792, -1.1225721836090088, 0.5945018529891968, 0.5666318535804749, 1.2337559461593628, 1.265332579612732, -0.3377048075199127, -2.040839433670044, -0.2504400908946991, 0.41000252962112427, 3.1742076873779297, -0.09942129999399185, -0.28106144070625305, 0.2351311296224594, 1.2915501594543457, 0.04413129389286041, -0.28689923882484436, -0.8175625205039978, -0.17867539823055267, -0.5600342750549316, -0.6429787278175354, 0.6151671409606934, -0.06208139285445213, 0.23666495084762573, -0.07326874136924744, 1.2014918327331543, 0.04284951463341713, 0.8760083913803101, 0.43603962659835815, 0.2575789988040924, 1.162099838256836, -0.7562513947486877, -0.30907514691352844, 0.7941270470619202, -0.9349842071533203, 0.4685203433036804, 1.0982245206832886, -0.8193150758743286, 0.8616784811019897, 0.047597046941518784, 1.7559446096420288, -0.5997782945632935, -0.7046738862991333, -0.48476335406303406, -0.6995005011558533, -0.02381332777440548, 1.6752468347549438, -0.9952786564826965, -0.8247436285018921, 1.8225395679473877, -0.24106310307979584, 0.2478708028793335, 0.4328559339046478, -0.05218768119812012, -1.7390012741088867, 1.8768301010131836, 0.9599754810333252, -0.4858035147190094, 1.2570335865020752, -1.2100034952163696, -1.3714191913604736, -0.8483083248138428, -0.8476567268371582, -1.3367908000946045, 1.1190884113311768, 1.1584968566894531, -1.4053692817687988, -0.7900837659835815, 1.54927659034729, 0.37350770831108093, -0.12492021173238754, 0.7226037383079529, 1.1924313306808472, 0.1538325846195221, -0.04643160477280617, -0.07233134657144547, -0.256435751914978, 0.6258944869041443, -0.30266204476356506, 0.22487202286720276, 0.10410690307617188, -0.05043011158704758, 1.3453665971755981, 1.3685672283172607, -0.22871339321136475, -0.7895984649658203, -0.13266362249851227, -1.0451582670211792, 1.114617109298706, -0.10493933409452438, -1.637457013130188, 0.8061168789863586, 0.1815718114376068, 0.9396799206733704, -0.033615414053201675, 1.5754508972167969, 0.17580237984657288, -0.14197610318660736, -0.5116907358169556], [1.505653977394104, 0.5226651430130005, -0.5235535502433777, 0.9944801926612854, -0.1280648410320282, -0.23882798850536346, -0.6447448134422302, -0.008038438856601715, -0.735141932964325, -1.005752682685852, 0.3317659795284271, -0.29272234439849854, 0.5112429857254028, -0.9078996777534485, -0.5487940311431885, 2.201411485671997, 0.31849777698516846, 0.12199272215366364, 0.44718247652053833, -0.24886618554592133, 1.597611427307129, 1.7374800443649292, 1.40897798538208, -0.9413772225379944, 0.3012961447238922, 0.21564260125160217, 0.27022209763526917, -1.1342989206314087, 0.8808097243309021, -1.075021505355835, -0.33725640177726746, -1.1093858480453491, 0.032322730869054794, -0.19686560332775116, 0.17263153195381165, 0.6931127309799194, -0.09127123653888702, 0.29527103900909424, -0.2596088945865631, 1.0238181352615356, -0.9576842784881592, 1.0085736513137817, 1.4946061372756958, -0.4422714114189148, -0.6738854646682739, 0.14142970740795135, -0.5284838080406189, -1.360012412071228, 0.3057558238506317, -0.5574589371681213, -1.1242952346801758, 0.7733148336410522, 0.1604401171207428, -0.5333753824234009, -0.38513240218162537, -0.38682979345321655, 1.107851505279541, 0.9152070879936218, -0.7702991366386414, 0.20128953456878662, 2.4508683681488037, 0.3005337715148926, 2.3760881423950195, 0.936787486076355, 0.99643874168396, 1.2067601680755615, 1.1805245876312256, 0.4241250157356262, 0.7805949449539185, 0.7098003029823303, 1.3564594984054565, -0.4169008135795593, -0.7826938033103943, -1.4624799489974976, 0.5198567509651184, -1.7922511100769043, 0.5876520276069641, 0.6180927157402039, -0.12419193238019943, 1.0466605424880981, -0.8996132016181946, -0.014035574160516262, -1.7368574142456055, -0.5294830799102783, 2.1139400005340576, 1.7585203647613525, -1.8368232250213623, 0.10436252504587173, -1.1634544134140015, 1.0868247747421265, -0.7481831908226013, 0.27612796425819397, -0.7739976644515991, -0.4296215772628784, -0.10016440600156784, -0.048866547644138336, 0.4256971478462219, 1.4932328462600708, 0.5179600715637207, 0.7319706678390503, 1.1807836294174194, 1.0189789533615112, -1.7329763174057007, 1.104844570159912, -0.3164655268192291, 0.37870267033576965, -0.20089782774448395, -0.9842734336853027, 3.1774682998657227, 0.12909355759620667, 1.7849208116531372, 1.5830135345458984, -0.5722571015357971, -0.47386661171913147, 1.3231747150421143, 0.9780135750770569, -0.6587474346160889, 0.5481091141700745, 1.1784400939941406, -0.3206664025783539, 1.8864408731460571, -0.9697659611701965, -0.07745782285928726, 0.8341966271400452, -1.6562241315841675, -0.3018907606601715, 0.7356958389282227, -0.814057469367981, -0.7526490688323975, -0.3260287344455719, 2.3421807289123535, 1.3224893808364868, -0.3878428339958191, 0.6431642770767212, 0.2799277603626251, -1.0531967878341675, -1.0934642553329468, -0.0004826711374334991, 1.6146327257156372, 0.2062552124261856, -0.22448918223381042, -0.988808810710907, 0.582298994064331, -0.9385963082313538, -0.13892969489097595, -0.8874439001083374, -0.8223558068275452, -0.006567864213138819, 1.8692466020584106, -0.14810600876808167, 0.033202286809682846, -0.627081036567688, 0.9997043609619141, -0.30458584427833557, 0.5542778372764587, -1.4972108602523804, -0.7562075853347778, 0.509751558303833, 0.6700974106788635, 1.0600394010543823, 0.2765776216983795, 2.154820680618286, -0.6123352646827698, 0.5599890351295471, 0.3320063352584839, -0.04702616110444069, -0.9362630248069763, -0.9782649278640747, 0.14595624804496765, 0.6420196890830994, 1.3815885782241821, -0.8188532590866089, 0.7323806285858154, -2.5030229091644287, -2.3272337913513184, 0.6573737859725952, 0.4708755612373352, 2.806591272354126, -1.1644974946975708, -1.2516924142837524, 0.24124807119369507, -0.03842882812023163, 1.6736512184143066, 2.240093469619751, 1.2676992416381836, 0.1780579686164856, 0.8440887331962585, 2.6296677589416504, 0.002567833289504051, 0.5574317574501038, -0.4438968598842621, -1.1989033222198486, 0.9871107935905457, 0.5841107964515686, -0.5807875990867615, -2.0804855823516846, -0.9815971851348877, -1.8689229488372803, 0.48505279421806335, 1.313462257385254, 0.39885514974594116, 0.578313410282135, 0.23358160257339478, 1.1552666425704956, -0.2807600796222687, 0.48502829670906067, 1.153782844543457, -0.2687430679798126, 0.7417015433311462, 0.19450487196445465, 0.6385483145713806, -0.9782150983810425, -1.2174854278564453, 1.3445948362350464, -1.503074049949646, 0.5393728017807007, -0.0019256609957665205, 0.8097292184829712, -0.49372053146362305, 0.3976520299911499, 0.44797283411026, 0.33844393491744995, -0.8476041555404663, 1.1851757764816284, -0.1863282471895218, 0.31492048501968384, -1.0548313856124878, 0.8453367352485657, -1.234416127204895, 0.4160117208957672, 0.5786991715431213, -0.3338153660297394, -0.04262624680995941, -0.39245733618736267, 2.5748443603515625, -0.2419145554304123, 0.33846229314804077, -1.1818714141845703, 0.10402748733758926, 0.31473493576049805, 0.8703013062477112, -0.25932493805885315, -0.24744096398353577, 1.7934235334396362, -0.7990769147872925, -0.7570034265518188, 0.13328735530376434, -0.9293032884597778, -1.8844612836837769, 1.443756103515625, -1.0561856031417847, -1.032279133796692, -0.18422992527484894, -0.6726758480072021, -1.4992763996124268, 0.3174505829811096, -0.5630307793617249, 0.5077552199363708, 1.2248477935791016, -0.43686485290527344, -0.11251939088106155, -0.6403809785842896, 1.6423810720443726, 1.2172539234161377, -0.6236570477485657, -1.1728320121765137, 1.0829133987426758, -1.6078481674194336, 0.3935035169124603, 0.08187741041183472, 0.3545074462890625, -0.6839351058006287, 1.272345781326294, -1.497218132019043, 0.27228373289108276, -0.15187886357307434, -1.2062338590621948, -2.0341603755950928, -0.6545522212982178, 0.9623887538909912, 0.6419403553009033, 0.1908518224954605, -0.5243468880653381, 0.361483097076416, -0.3861981928348541, 0.45030343532562256, -0.8588857054710388, 0.23627479374408722, 1.0044639110565186, -0.6006214022636414, -1.3868134021759033, -1.1124305725097656, -0.2569717466831207, -0.3873685896396637, -1.0947710275650024, -0.528796374797821, -0.16110797226428986, -0.2335216999053955, 0.7005185484886169, 0.1606421172618866], [-0.9225317239761353, -0.19428125023841858, 1.2050518989562988, 1.5910755395889282, -0.24365465342998505, 0.6606016755104065, 0.3361181616783142, 1.5366795063018799, -0.9565575122833252, 0.677160382270813, -0.269445538520813, -0.1631791591644287, -1.7352584600448608, 1.3273231983184814, 0.2596540153026581, -0.7183985114097595, 0.6049462556838989, 0.18280568718910217, -4.287429332733154, -0.3689883053302765, -0.9070965051651001, 0.25664156675338745, -0.9603700637817383, 0.21050746738910675, -0.10158724337816238, -0.5838509798049927, -0.6324005126953125, -0.9829816222190857, -2.2393932342529297, -0.35287952423095703, -0.40579551458358765, -0.7410779595375061, -2.6495742797851562, -0.11465020477771759, -0.80635005235672, 0.8567699790000916, 0.4612380564212799, 0.26555272936820984, -0.1448259800672531, -1.2553553581237793, 0.2198018878698349, 0.808327853679657, -0.013680468313395977, 0.433289498090744, -1.2452608346939087, 0.8653541207313538, 0.5642536878585815, 0.9938018918037415, -0.17455075681209564, -0.45845741033554077, -0.6737381815910339, -0.7483277916908264, -2.6517837047576904, 1.4558005332946777, -0.26701608300209045, -0.2728627920150757, -0.4139820635318756, 0.515877366065979, -1.5607777833938599, 0.6234328746795654, -1.5412983894348145, 0.9213375449180603, -0.3452570140361786, 0.46176084876060486, -1.3999258279800415, -2.2499351501464844, 0.24556981027126312, 2.224583387374878, 2.1175806522369385, 0.08418963849544525, 2.8822011947631836, 0.9955302476882935, 0.8502452969551086, 0.07074445486068726, -1.1692768335342407, 0.08196187764406204, 0.7701491713523865, 2.5576765537261963, -1.9101386070251465, -2.2544541358947754, -0.31904953718185425, -0.5696839094161987, -0.2833012044429779, -0.2588190734386444, 0.5269128084182739, -0.5122731924057007, 1.3860812187194824, -1.769417405128479, 1.0629206895828247, -1.4637610912322998, 2.0273303985595703, -1.9651073217391968, 0.6628894805908203, 0.9961209297180176, -0.6007536053657532, -1.2862136363983154, -1.4811729192733765, -0.07804452627897263, 0.9045335650444031, -1.394999623298645, -0.732498288154602, 0.4830564856529236, 0.7998377084732056, -1.6491186618804932, -0.4813573658466339, 1.1808648109436035, 1.4435878992080688, 0.8088496327400208, 0.12754857540130615, -0.21484357118606567, -0.17090824246406555, -0.15966865420341492, -2.3192479610443115, 1.7256770133972168, 1.3267250061035156, 0.4546376168727875, -0.29252997040748596, -1.8055332899093628, 0.5124430060386658, -0.6461156606674194, -0.458588570356369, 0.9196725487709045, -1.1714801788330078, -0.14128945767879486, 0.18088792264461517, 0.7839325070381165, -0.7938252687454224, 0.8793063163757324, 0.6916238069534302, 1.7420607805252075, 0.8346483111381531, 0.20934642851352692, 0.186686709523201, -1.1400443315505981, -0.05673317611217499, 0.48079055547714233, -1.8845546245574951, -0.1458447277545929, 3.007371187210083, -1.3211077451705933, 0.4259076416492462, 1.1969001293182373, -0.6043034791946411, -0.2855599820613861, 0.18209099769592285, -1.3194468021392822, -1.1121476888656616, 2.0501232147216797, 0.8066661953926086, 1.4384056329727173, -0.043941520154476166, -1.0863704681396484, 1.8824814558029175, 1.7516968250274658, 0.5060597658157349, -3.3378236293792725, 0.37579545378685, -0.14024437963962555, 0.691161572933197, 0.1958167403936386, -2.3203604221343994, 0.7859681248664856, 0.04842330142855644, -0.9249109625816345, -0.2432524561882019, -0.3651061952114105, -0.6448917984962463, 1.11649751663208, 0.6163773536682129, 1.8409757614135742, -0.7737358212471008, -2.3814849853515625, -1.3372572660446167, 0.2471752166748047, -0.17347538471221924, -1.2864941358566284, -1.1506052017211914, 0.5221303105354309, -1.7727175951004028, 0.9843050241470337, -1.4756876230239868, 0.4713706374168396, 0.019175129011273384, 0.5383095741271973, 0.7904878854751587, 0.02270505018532276, -0.6033925414085388, -1.0112507343292236, -0.3845764994621277, 0.6934950351715088, 1.3386850357055664, 0.02148958295583725, -0.5072595477104187, -0.4157385528087616, -0.5737345218658447, -1.0963472127914429, -0.12660326063632965, -0.9412462711334229, 0.5346977710723877, -0.2894801199436188, -0.4801308512687683, 1.9582287073135376, -0.8028154373168945, -0.5882169604301453, -0.7064831256866455, 0.5208480954170227, -0.4378390908241272, -1.7712419033050537, 0.24389177560806274, -0.23888951539993286, -0.1665802299976349, -1.2856814861297607, -0.32966265082359314, 0.4047118127346039, 0.8843137621879578, -0.8339142203330994, -1.1236244440078735, 0.6890310645103455, 0.4626431465148926, -0.7961845993995667, 0.15144626796245575, 1.8013406991958618, -0.7994353175163269, -0.8077131509780884, -0.024682920426130295, 0.22203126549720764, 0.3323541581630707, -1.8257580995559692, -1.0446511507034302, 1.770377278327942, 0.48366832733154297, -1.627824068069458, 0.6483418941497803, -0.3702840209007263, -0.41285330057144165, -1.4496508836746216, 0.5859842300415039, 0.9839655756950378, 0.35236623883247375, 0.3527965843677521, -0.5863589644432068, 1.205815315246582, 1.7866028547286987, -1.5401480197906494, 0.07273630052804947, -1.1199232339859009, 0.5614966154098511, 0.6718840003013611, 3.01942777633667, -0.036994222551584244, 0.5145754814147949, 0.3077666759490967, 0.3514496088027954, 0.8804619908332825, -0.11892097443342209, -0.7752828598022461, -0.37266191840171814, 0.24987675249576569, 0.2366311252117157, 1.249403953552246, -0.10225427150726318, -1.4565770626068115, 0.25195491313934326, -0.7297855615615845, -0.49700498580932617, -0.5386837720870972, 0.6117575764656067, -1.0964372158050537, -1.6569486856460571, -0.3339628577232361, -0.30286017060279846, 0.12747715413570404, -0.9425861835479736, -0.32057374715805054, 0.6598383188247681, 0.5860592126846313, -1.2064483165740967, -0.5963294506072998, 0.31587401032447815, -1.2350637912750244, 0.4007106125354767, -0.6418020129203796, 0.15556959807872772, 0.8471981883049011, -1.2382969856262207, -1.2228513956069946, -1.740706205368042, -0.45917609333992004, 1.6080682277679443, -1.5063101053237915, 0.7224783301353455, -0.5322666764259338, 0.5440143942832947, 0.01341771800071001, 0.47445613145828247, 2.319899082183838, -1.502543330192566, -0.352857768535614, -0.8757688403129578, 1.1479272842407227], [-0.180612713098526, 0.5560905933380127, 0.35215821862220764, 1.330199122428894, -1.4333455562591553, -1.6563794612884521, 0.4357382655143738, -0.9185937643051147, -0.1428830772638321, -1.4874588251113892, 0.6744340658187866, 1.3962842226028442, -0.8783559799194336, 1.3431206941604614, -0.23213067650794983, -0.13144974410533905, -0.890934944152832, -0.44575342535972595, -0.40027377009391785, -0.6506105661392212, 1.2521357536315918, -1.2283918857574463, 0.2019018977880478, -1.0930159091949463, 1.8980813026428223, -0.40651318430900574, -0.05411620810627937, 2.00537371635437, -0.5175749063491821, -0.19302892684936523, -0.007038705516606569, 0.03231751173734665, 0.5100383758544922, -0.7636661529541016, -0.26554709672927856, 1.3525229692459106, 1.5637840032577515, 0.2011624425649643, -0.13733115792274475, -0.7848740816116333, -0.1772100180387497, -0.70925372838974, -0.47464442253112793, 0.7980024814605713, -0.2729254961013794, -0.3132610619068146, 0.7777636051177979, -1.1970081329345703, -2.370920419692993, 0.9344644546508789, 1.1520494222640991, 0.6091831922531128, 0.5372653007507324, -0.8500697016716003, 0.6015048027038574, -0.6074591875076294, 0.2353777438402176, 0.7238485813140869, 0.7028045654296875, -1.1880677938461304, -1.4344416856765747, 1.7267677783966064, 0.6706679463386536, -1.3107662200927734, 1.8373786211013794, 1.489322304725647, -0.20540538430213928, 0.7299135327339172, -0.02831602282822132, -0.49417850375175476, 2.1007447242736816, 0.23811280727386475, -2.254770517349243, 0.9395864605903625, -0.3271942734718323, -0.49293580651283264, 1.1253541707992554, -0.4065994918346405, -0.3197941780090332, 2.155954122543335, -1.0407060384750366, 0.007267102599143982, 0.8665111660957336, -0.02256493642926216, -3.0425643920898438, 0.6503869295120239, -0.0323420949280262, -1.1619927883148193, -0.5460286140441895, -1.4772405624389648, -0.3497024476528168, 0.5115667581558228, -0.5353939533233643, -0.0234686266630888, 0.1685289889574051, 1.1161197423934937, 0.29667994379997253, 0.7180551886558533, -0.11259593069553375, -1.2489538192749023, -1.0906033515930176, -0.7497513890266418, -1.1592872142791748, -0.13718198239803314, 0.06970806419849396, -0.09756623953580856, -0.15657486021518707, 0.546621561050415, 1.087571144104004, 1.3304319381713867, -0.07118964195251465, -1.7612987756729126, -2.75539231300354, 0.7172211408615112, -0.20033621788024902, 0.504517674446106, -2.1171889305114746, -1.8323664665222168, -0.7931638360023499, 0.8950912952423096, 0.9907897114753723, 0.9748611450195312, -1.158319354057312, 2.0418496131896973, 0.725051999092102, -0.554806113243103, 0.415352463722229, -0.1889025866985321, 0.9180828332901001, 0.42615360021591187, 3.235306739807129, 0.5785025954246521, -0.5774719715118408, 0.0877520814538002, -0.03520214185118675, -1.4257856607437134, -0.47042763233184814, -1.1999242305755615, -0.2407657504081726, -0.8539867401123047, -1.1592124700546265, -0.0509917326271534, 1.0047041177749634, -0.02815832570195198, 0.1252286732196808, -0.75661301612854, 1.3982940912246704, -0.6795430779457092, -0.5956761240959167, 0.8440529108047485, 1.3824843168258667, 0.3521248400211334, -1.2207670211791992, -1.1966909170150757, -0.77254319190979, -0.3796890377998352, 0.2607375681400299, -0.7361930012702942, -1.5502654314041138, 0.5930723547935486, -0.47635555267333984, 0.7599400281906128, 0.576943039894104, 0.22531460225582123, 0.5304300785064697, 0.8369730114936829, -0.14885364472866058, 1.8391437530517578, 0.4857621490955353, -0.6132352352142334, -0.16444845497608185, 1.1561436653137207, -1.7750886678695679, 0.19148191809654236, 0.014694422483444214, -0.9233185648918152, -0.34524786472320557, 0.0925624892115593, 0.7428662180900574, -0.09674453735351562, -1.6334511041641235, 1.1399850845336914, -0.3246208131313324, -1.3774672746658325, 0.8179855346679688, 0.04495849460363388, 0.11319614946842194, 0.6258991956710815, 1.191580891609192, -1.2059366703033447, -0.8242167234420776, -0.40291181206703186, 1.0178327560424805, -1.0406304597854614, -0.6083505749702454, 0.05632519721984863, -1.0077394247055054, 0.26037588715553284, 2.0844805240631104, 0.7248611450195312, -0.6856194734573364, -0.8991827964782715, 0.7844117879867554, -0.13126115500926971, -0.6689512729644775, 0.6554754376411438, 0.7356972694396973, 0.3153704106807709, 0.32020920515060425, -2.01080322265625, -0.8627417683601379, -1.7042468786239624, -1.818394422531128, -0.7223654389381409, 0.8153151273727417, 0.7842870950698853, 0.5178982019424438, 0.22124943137168884, -0.8456968069076538, -1.034791111946106, 0.2621367275714874, -1.1699750423431396, -0.5541430115699768, 0.9053870439529419, -3.2429213523864746, 0.35313382744789124, 0.6227303743362427, 1.6728233098983765, -0.2931066155433655, 3.078939437866211, -0.2660370171070099, -0.5906267166137695, 1.410140872001648, 0.6158950924873352, -1.8770853281021118, 0.055416353046894073, 1.5256710052490234, 0.1238662376999855, -0.2806497812271118, 1.1454805135726929, 0.5126387476921082, -0.2967444658279419, -1.584779143333435, 0.18801963329315186, 0.5344845652580261, -0.6086608171463013, 0.594048023223877, 0.05580168962478638, 0.43819934129714966, -0.35985061526298523, 0.3015601634979248, 1.1044206619262695, -1.3162401914596558, 0.0697222501039505, 2.0502092838287354, 0.24081756174564362, 0.953875720500946, 0.20088203251361847, 1.1034806966781616, 0.6512986421585083, -0.5324047207832336, 0.6036109328269958, -0.2522932291030884, 1.0947833061218262, -0.9106362462043762, 0.05169595405459404, 0.2607978284358978, -1.604445219039917, 1.3825620412826538, -0.3197266161441803, -2.531843423843384, 0.8168929815292358, -0.9518961906433105, -1.047998309135437, -2.212357521057129, 0.8886739015579224, 0.010372316464781761, -0.7604414224624634, 0.4247894287109375, 0.04752250388264656, 0.134079709649086, 0.7702665328979492, 1.3834887742996216, -0.22659727931022644, -0.4239211976528168, 0.3326110243797302, -0.24772323668003082, -0.274515300989151, -0.5623327493667603, -0.412732869386673, 0.4640146791934967, -0.041490983217954636, 0.5763159394264221, 0.6678079962730408, 0.5567992925643921, 0.4633409082889557, -0.5056827664375305, 0.14144432544708252, -0.21476620435714722, 0.45768648386001587], [0.6842892169952393, -1.078655481338501, -0.013616534881293774, 1.3479894399642944, 0.6236613988876343, -0.5951399803161621, -1.4452298879623413, -0.6351386308670044, -0.24532552063465118, 0.5507611632347107, -0.39947208762168884, -0.7679237723350525, 0.9367847442626953, 0.5547308921813965, -0.6328311562538147, 0.04325375705957413, 0.3728480637073517, -0.4959199130535126, -1.5486799478530884, 1.1315408945083618, -0.14794915914535522, -0.24421656131744385, 0.2196175456047058, 0.3771106004714966, 0.594985842704773, 0.9983324408531189, 1.6886546611785889, -0.6444300413131714, -0.2698498070240021, 0.23524849116802216, 0.3860098123550415, -0.3430560827255249, 0.03146931156516075, 0.15553319454193115, 0.6374614238739014, -2.9258906841278076, -0.7976150512695312, -0.5390186905860901, -1.5679569244384766, 1.4959092140197754, -1.8854379653930664, -0.38992545008659363, -0.6693536043167114, -0.30427679419517517, 0.49717870354652405, 1.6962765455245972, -2.1232264041900635, 0.45588886737823486, 0.235505610704422, -0.5426910519599915, 1.0858299732208252, 1.8396861553192139, -0.6743595600128174, -0.018605340272188187, -0.18419522047042847, 1.6001566648483276, 0.0020210412330925465, 1.3891185522079468, 0.039022911339998245, 1.9084957838058472, -0.047314636409282684, -0.3953857719898224, 0.34457671642303467, 1.4740451574325562, 1.30583918094635, -0.3185444176197052, 0.7652472257614136, 1.2498201131820679, 1.1109172105789185, 0.23348410427570343, 0.8679327368736267, 0.8246453404426575, -1.4505921602249146, 0.034088894724845886, 0.7796033620834351, 0.7709391713142395, 0.29430681467056274, 1.0732922554016113, 2.929513454437256, -0.12091585993766785, -0.3275400996208191, -0.3005432188510895, -0.4094223976135254, -0.570999026298523, -0.28494536876678467, -1.5900248289108276, 0.13646644353866577, 0.07977612316608429, 0.8966218829154968, 0.7518897652626038, -1.2160738706588745, -0.10294391959905624, 0.17871826887130737, -0.14202475547790527, 0.8661336898803711, 1.7982450723648071, 1.516278624534607, -0.4590352177619934, -0.03256538137793541, 0.21966157853603363, 0.21342577040195465, -1.1698658466339111, 0.2749161720275879, -1.6355900764465332, 0.5456275939941406, -1.5788806676864624, 0.4447652995586395, 0.05584820732474327, 1.2785133123397827, -0.3802746832370758, 0.4829448461532593, 0.43759796023368835, -1.021972417831421, 1.5049419403076172, 0.6367515325546265, 1.812031626701355, 0.844922661781311, -0.4463347792625427, 0.7699018120765686, 1.1449352502822876, 0.16749607026576996, 0.20608492195606232, 1.1800943613052368, -0.18160322308540344, -2.719904661178589, 0.4417128264904022, 0.19033989310264587, 0.016375193372368813, 0.5741134285926819, 0.15443909168243408, -0.13700726628303528, -1.4455970525741577, -0.4608089327812195, -2.7944488525390625, -0.4131424129009247, 2.360025405883789, -0.5154410004615784, 0.33054786920547485, -1.0283321142196655, 1.2320460081100464, -0.7608374953269958, -0.9728808999061584, -1.059806227684021, -0.7798171043395996, -1.0060431957244873, 0.46366941928863525, 1.9222235679626465, -1.0356074571609497, 2.2433900833129883, -0.04448799788951874, -0.7197080254554749, -0.5842968225479126, -0.2580774128437042, 0.30569520592689514, 0.4483225643634796, -0.2685094177722931, -0.2868097126483917, 2.037562370300293, -1.0099985599517822, -0.6402387022972107, -0.49422869086265564, 1.4971884489059448, -0.6480231285095215, -1.2938685417175293, 1.7485177516937256, -1.1850440502166748, -1.0179835557937622, -0.6658182740211487, -0.7841938138008118, -0.36236119270324707, -1.8809794187545776, -0.2764403223991394, 1.1289364099502563, -1.1400161981582642, -1.134669542312622, 1.661893606185913, 1.0939809083938599, 0.04674213379621506, -1.6113522052764893, 1.4283422231674194, 0.21356134116649628, 0.50454181432724, 0.4234209358692169, -1.45184326171875, -0.5841478705406189, 1.1608859300613403, -1.1460384130477905, 0.25536373257637024, -0.5693708658218384, 0.09764724224805832, -2.1543617248535156, -0.7449836730957031, 0.08242753893136978, 1.8342688083648682, 0.003792624222114682, -0.020316779613494873, -0.3201533854007721, 0.14985984563827515, -1.684251070022583, -0.920563817024231, -0.17295195162296295, 0.4385661482810974, -0.9782084226608276, -0.14676858484745026, -0.532452404499054, 1.8959417343139648, -1.2959215641021729, 0.23170363903045654, -0.7409881949424744, -0.9550275206565857, 0.46591946482658386, -0.1263711303472519, 0.5817652940750122, 0.13778164982795715, 1.2941502332687378, 2.558450222015381, 0.21893641352653503, 0.7437250018119812, -0.776399552822113, 0.45136749744415283, 1.6865922212600708, 0.004578573163598776, 1.027379035949707, -0.5594967007637024, -0.9946578145027161, 0.9521874785423279, 1.2240304946899414, 0.9262266755104065, 0.17684128880500793, -1.751619577407837, -0.3037925064563751, -0.7570685744285583, 0.05032689869403839, -0.003562652738764882, -0.7672114372253418, -0.06163959577679634, -0.35845300555229187, -1.663264274597168, -1.6209297180175781, 0.23310814797878265, 0.5342798829078674, 0.22069211304187775, 1.2745380401611328, -0.6391116380691528, -0.3163769543170929, -1.3877496719360352, -0.44784194231033325, -0.0015728622674942017, 1.7285417318344116, 0.35273513197898865, 1.8571677207946777, -1.3390696048736572, -0.3162252604961395, 0.6154823899269104, 0.495432585477829, 0.31473180651664734, -0.5499887466430664, 0.9180112481117249, 0.5278854370117188, 0.5568279027938843, -0.21628060936927795, 0.41892531514167786, 0.21018853783607483, 1.094429612159729, 0.43648988008499146, 1.3919111490249634, 1.3306468725204468, 0.9578408002853394, -0.5396844744682312, -0.3694082498550415, -1.1140918731689453, 0.7972288727760315, 0.42866408824920654, 0.49649861454963684, 1.4025768041610718, 0.9722588062286377, 1.4546713829040527, -1.5987838506698608, 0.7414141297340393, -0.2730388045310974, 0.549665093421936, -0.9902079105377197, -0.5509871244430542, 1.1109557151794434, 0.3629956841468811, 2.064514398574829, -1.4918603897094727, -0.27451175451278687, 0.3535815477371216, -0.560888409614563, -0.3419248163700104, 0.918228805065155, -0.8636025786399841, -0.7338658571243286, 0.2059364914894104, 0.5360972881317139, -0.730030357837677, 1.1321772336959839, -0.6038833856582642, -0.5541014671325684], [-0.8144890069961548, -0.022851835936307907, -0.5450899600982666, -0.48062294721603394, 0.16862261295318604, 0.23903696238994598, 0.551602303981781, 0.6205645203590393, 0.12185343354940414, 0.39683181047439575, 0.5139093995094299, -1.3030582666397095, 0.3278607130050659, -0.025657974183559418, 0.27497726678848267, 0.12025409191846848, -0.025085749104619026, 2.1329433917999268, -1.110613226890564, 2.6231958866119385, 1.2535793781280518, -0.39199861884117126, -1.5716122388839722, 0.28485479950904846, 0.07701841741800308, 0.5077163577079773, -0.16692736744880676, -0.19578975439071655, 1.8359978199005127, -0.4033183455467224, -1.2655209302902222, -0.2622581720352173, 0.7477210164070129, 0.3768441379070282, 0.06538838893175125, -0.35625845193862915, -0.04227754473686218, -1.220909595489502, -1.0224329233169556, 1.0059126615524292, -1.166429877281189, 0.9526142477989197, 0.3818839192390442, -0.6890357732772827, 0.1602807343006134, -0.28079721331596375, 0.11802174150943756, -0.027626903727650642, 0.07789517939090729, -2.7016639709472656, -0.2944605350494385, 1.3708006143569946, -1.8742567300796509, -0.27001550793647766, -0.07286527752876282, 1.9732608795166016, -0.414824515581131, 1.0707712173461914, -0.05443127453327179, -0.020268550142645836, -0.7722811102867126, -1.5856926441192627, 0.5493788123130798, 0.720201849937439, -1.138067364692688, -0.9743544459342957, -1.235112190246582, 0.4060456156730652, -1.2912306785583496, -0.8407622575759888, -0.6465342044830322, -0.46203410625457764, 0.8510050773620605, 0.4894600808620453, 0.4371660351753235, -0.13870356976985931, 0.6075908541679382, 0.8491837382316589, -1.0552634000778198, -1.3291348218917847, 1.2398914098739624, 0.35675421357154846, -0.9773322343826294, 0.7309567928314209, 0.4471578598022461, 0.21196302771568298, 0.409013956785202, -0.43596047163009644, -1.525066614151001, 1.9580599069595337, -0.2047552466392517, -1.4189436435699463, -0.9313380122184753, -0.26624423265457153, -1.685334324836731, -0.813381016254425, 1.8120988607406616, 0.19325391948223114, 0.9020490646362305, -0.18591849505901337, 1.2394102811813354, 1.3874437808990479, 1.009060263633728, 0.13138966262340546, -0.3316527009010315, -0.11850160360336304, -0.9521889090538025, 1.2143293619155884, 0.5018073916435242, 0.6921490430831909, 0.46516653895378113, -0.1523803174495697, -0.6730293035507202, 0.6627082228660583, 0.9110623002052307, -0.7866950631141663, 1.3217840194702148, 1.3875921964645386, -0.6967580318450928, -0.926896870136261, 0.4074687361717224, 0.7216254472732544, 1.1984992027282715, 0.6997879147529602, -1.2167751789093018, 1.4867925643920898, 0.4553292989730835, 0.1458711475133896, 0.3386954963207245, -0.3285798728466034, -0.5393123626708984, -1.4539473056793213, -1.8186790943145752, 1.1840534210205078, 0.3780604600906372, -2.648524045944214, 0.12397214770317078, 0.934851884841919, -0.9940049052238464, -1.489862084388733, -2.1803691387176514, -0.22723205387592316, -0.8394734263420105, 0.23088061809539795, -0.7656779289245605, -1.63302481174469, 0.8233866095542908, 0.6148451566696167, -1.449924349784851, 0.108177050948143, 1.6108754873275757, 1.4875304698944092, -0.008553741499781609, 0.3016156852245331, 1.2098928689956665, 0.7002134323120117, 0.6514760851860046, 0.30352866649627686, 0.5289164185523987, 0.6170670390129089, 0.2150658816099167, 1.3509204387664795, 0.08561857789754868, 0.19141808152198792, -0.3244871497154236, 0.6893913745880127, -1.482984185218811, 0.05389288812875748, 0.9353511929512024, 1.3496613502502441, 0.5922725200653076, -0.8704302906990051, 1.802442193031311, 0.3681979179382324, 0.8584887981414795, -0.5504647493362427, -0.42549723386764526, -1.5069011449813843, -0.449674129486084, -0.15035594999790192, -0.6512286067008972, -0.398623526096344, -1.2133797407150269, 0.5057884454727173, 1.2268760204315186, 0.47871577739715576, 1.5625433921813965, -0.09287945181131363, 0.7953150272369385, 0.1892583817243576, 0.7533321976661682, -0.6856154203414917, 0.6868401765823364, 0.0705704316496849, -1.0369123220443726, 0.8450894355773926, 0.1998845338821411, 0.34901320934295654, 0.16627132892608643, 1.4981447458267212, 2.5390477180480957, 1.003178358078003, 2.0165698528289795, 0.012227662838995457, 0.48636937141418457, 0.6223371028900146, -0.0797828882932663, -1.945855975151062, 0.38870662450790405, -2.1100194454193115, -1.929473638534546, 1.4880380630493164, 0.4225764572620392, 0.9047303199768066, -0.8774449825286865, -0.34560996294021606, -0.03157704696059227, -1.831240177154541, -1.0716840028762817, 0.1471433788537979, -0.4503856301307678, 0.9119523167610168, -0.11856239289045334, 1.2710487842559814, -1.9596461057662964, -0.3449452519416809, -0.18761086463928223, -1.021490454673767, -1.2418092489242554, -1.9374998807907104, 0.3887850046157837, 1.0725312232971191, 1.0961462259292603, 0.9460872411727905, 1.5090073347091675, -0.3280709981918335, -0.19320595264434814, 1.7569514513015747, 0.1417180299758911, -1.0842761993408203, -1.0360746383666992, 1.8443262577056885, 0.16768276691436768, 1.0524204969406128, -0.9706059694290161, 0.34436264634132385, -0.4308897852897644, 1.0534899234771729, 0.16988061368465424, 1.4904162883758545, 0.5090603232383728, -0.308521032333374, 0.4475817084312439, 0.6598698496818542, 0.13803383708000183, -0.22477120161056519, 0.7019566297531128, 0.7202818989753723, -0.7785996198654175, -0.3794926404953003, 0.6359472274780273, 1.2289847135543823, 0.02678450010716915, -0.8052374124526978, -0.7781863212585449, -0.7278135418891907, -1.3474572896957397, -0.05760483071208, -0.7746901512145996, 0.11232634633779526, -0.12619349360466003, 0.4882054626941681, 0.9572020173072815, 0.00832934770733118, -1.2173088788986206, 0.13307060301303864, 0.3351414203643799, -0.02416141703724861, -0.6201267838478088, 0.4715937077999115, -1.0415666103363037, 0.03462115675210953, -0.9130024313926697, -0.7988981008529663, 0.024155663326382637, 1.0845670700073242, -1.2046903371810913, -0.18147487938404083, 0.36724281311035156, -0.7883236408233643, -0.1233404129743576, -1.0968776941299438, 0.6017829775810242, 0.9352251291275024, -1.305197834968567, 0.8532021045684814, -0.09429284185171127, 0.18671558797359467, -0.7213088870048523, -0.07025683671236038]], "dtype": "float32"}, {"__nd__": [[-1.1077203750610352, 2.1031758785247803, 0.29951807856559753, 0.3067163825035095, -1.5189579725265503, 1.4161211252212524, 0.08547037094831467, -0.40491783618927, -0.06612575054168701, -0.043054912239313126, 0.1883493810892105, -0.27172476053237915, -2.179466962814331, 0.5504377484321594, 0.34075456857681274, 0.2623288631439209, 0.23764823377132416, 0.7540946006774902, 0.9766178727149963, 0.5707709789276123, 0.6528851985931396, -1.0759186744689941, -0.5192602276802063, -1.441873550415039, -0.9400626420974731, -0.7856699228286743, 1.4128285646438599, 0.05851681903004646, -0.22825221717357635, 0.6234950423240662, -0.37494874000549316, -0.20457002520561218, -1.0634616613388062, 0.7237409353256226, 0.7441027760505676, -0.37676000595092773, 0.08962253481149673, -0.435926616191864, -0.06473398208618164, -0.949704110622406], [1.8376057147979736, 0.8576862812042236, 0.5186285376548767, -3.0308001041412354, 0.2229442447423935, -1.036195993423462, 1.8203648328781128, -0.36421194672584534, 0.8628836870193481, -0.32185831665992737, 0.3579632341861725, -1.2011568546295166, 0.8166369795799255, -0.7991949915885925, 1.215266466140747, 0.9130545854568481, -2.10341477394104, -0.27178171277046204, -0.2983019948005676, -0.41446614265441895, 0.4228702187538147, -0.24696098268032074, -0.2975851595401764, -1.99471914768219, -0.4246068596839905, 0.48573553562164307, -0.5516440868377686, -1.0733208656311035, 1.1018483638763428, -2.4485018253326416, 1.7521675825119019, -0.48114192485809326, -0.9778980612754822, -0.36881279945373535, -0.8220959305763245, 0.9822337031364441, 1.355713963508606, 0.039127420634031296, -0.3608628213405609, 0.3413552939891815], [-0.7927160859107971, 0.7740584015846252, 1.6731637716293335, -0.49320369958877563, -0.2652030289173126, -0.17398393154144287, 1.502609133720398, 2.058316469192505, 2.0669784545898438, 0.3748905062675476, -1.6440634727478027, -1.2742905616760254, 0.5916587710380554, 1.2656986713409424, -0.8914377093315125, 1.8470807075500488, 1.7382543087005615, -1.2506420612335205, 0.30667516589164734, 0.17346882820129395, 1.0202796459197998, -0.09525732696056366, -0.14495258033275604, 0.8203660249710083, 1.0827317237854004, 0.6427581906318665, -0.3793849050998688, -2.3303446769714355, 0.4938075840473175, -0.08959795534610748, -0.7435120940208435, -0.32465222477912903, -0.34444963932037354, -1.5569360256195068, 0.1634567826986313, 0.4060940444469452, -2.7299463748931885, 2.1250970363616943, -0.5818162560462952, 1.2135001420974731], [1.0060943365097046, 1.46247136592865, -0.14227698743343353, -0.039965324103832245, -1.7575840950012207, 1.0898607969284058, 0.8812622427940369, 0.16882824897766113, 0.527083158493042, 2.7434113025665283, -0.043626584112644196, 0.022533129900693893, 0.14702355861663818, 1.328100562095642, -1.9309569597244263, 1.148762822151184, -0.2014768272638321, -0.6155656576156616, 0.7298892736434937, 0.20201250910758972, 1.5184762477874756, -1.869805932044983, 0.643467366695404, 1.0285173654556274, 0.8652560710906982, -1.3315083980560303, 0.5174133777618408, 0.41138380765914917, -0.24255144596099854, -0.49027904868125916, 1.2928887605667114, -0.5124335885047913, 0.8209244012832642, 0.7515329718589783, 1.4730879068374634, -1.1557546854019165, 1.1099082231521606, 1.4818775653839111, -0.8407219648361206, -0.26248615980148315], [-0.4363501965999603, 0.17934182286262512, 0.20538653433322906, 1.0065739154815674, -0.338695228099823, 1.3584849834442139, 1.1499648094177246, -1.5295608043670654, -1.018875002861023, 2.425452470779419, 1.2327800989151, 0.4823830723762512, -0.5825733542442322, 0.8281958699226379, -0.5435163974761963, 1.1721173524856567, 1.3176578283309937, -0.01622735522687435, -2.0153756141662598, -0.8291853666305542, 0.2901704013347626, -0.13168494403362274, -0.25774499773979187, 1.8563259840011597, -0.1635059267282486, 0.4650896489620209, -0.9117544293403625, 0.5708910226821899, 0.18440262973308563, 0.05189555138349533, -0.6486657857894897, 0.5378659963607788, -0.6335703134536743, -0.4792856276035309, 1.344663381576538, 1.1122480630874634, -0.4145902991294861, -0.8514466285705566, -0.7345743179321289, -0.37412238121032715], [-2.286297082901001, -0.29982638359069824, -0.2770809531211853, -1.4862451553344727, -0.8200896978378296, 0.4765304625034332, 0.32593482732772827, 1.6710923910140991, 1.1415741443634033, 1.0038328170776367, -1.274079442024231, 0.03281974047422409, -0.8185251355171204, 0.5667349100112915, -0.44638070464134216, 0.49829888343811035, -1.0785863399505615, 1.1387373208999634, -1.2978590726852417, 1.464606761932373, -0.45897582173347473, -0.5951845645904541, -2.1491312980651855, 0.07229084521532059, 0.04454703629016876, 1.5828181505203247, 0.4875466823577881, -2.2152116298675537, 0.35887035727500916, -0.2038915604352951, -0.6847891211509705, 0.9782261252403259, -0.1885310411453247, -0.180201455950737, 1.4594968557357788, -0.08257067948579788, 0.4943491518497467, 2.475945234298706, -0.21396733820438385, -0.11749403178691864], [0.720162570476532, 0.9139524102210999, -0.0846916139125824, -1.0649727582931519, -0.7311950922012329, 0.5240283012390137, 0.46252891421318054, -0.4377508759498596, -1.118553638458252, 0.1523565948009491, 1.107631802558899, -0.5174216032028198, 0.35324549674987793, 0.5568972826004028, -2.2051424980163574, 1.4889005422592163, 1.648841142654419, 0.8980345129966736, 0.3410900831222534, -0.38699570298194885, 1.1859787702560425, 0.8150672316551208, 1.7312813997268677, 0.29577895998954773, 0.8790603280067444, 0.6118839383125305, -0.3624422550201416, 1.3421721458435059, 0.5498203039169312, 0.1008741483092308, 1.4263579845428467, 1.7599730491638184, 1.3194044828414917, -0.03998225927352905, -0.713523268699646, -0.1291329562664032, -0.039210572838783264, 1.1723488569259644, -0.5786944627761841, -0.45730799436569214], [1.1500686407089233, -1.3033673763275146, 0.1516800969839096, 1.8182705640792847, 1.4185208082199097, -0.16220404207706451, 0.17530381679534912, 0.2171836644411087, 0.6995474696159363, 0.2619675099849701, 0.9228760004043579, -0.1216917484998703, -0.09232168644666672, -0.9930053949356079, 0.09802792966365814, -0.6681997179985046, -0.8965669274330139, -1.1680831909179688, 1.823934555053711, 0.7140054702758789, 0.6921154856681824, -0.3035559058189392, 0.8069490194320679, -0.42838358879089355, -0.5995603799819946, -0.5021252632141113, 0.03392957150936127, -0.30218714475631714, 1.0840167999267578, -0.5260365009307861, 0.5801160335540771, 0.008843347430229187, 0.9948882460594177, 1.1947529315948486, -1.7792834043502808, -0.4541592597961426, 1.7071200609207153, -0.44368505477905273, 2.342013120651245, -1.1250492334365845], [-2.3005881309509277, -1.5804253816604614, 1.1227754354476929, 0.9190608859062195, -0.09620780497789383, -1.238925814628601, 0.028718343004584312, 1.580880880355835, -1.124561071395874, 1.2289705276489258, -1.7358592748641968, 1.0178484916687012, -0.4297448396682739, -1.6826684474945068, 1.0916519165039062, -0.30125102400779724, -0.7867239117622375, -0.4074992835521698, 0.13152338564395905, -2.155665397644043, 0.3359126150608063, 1.4765636920928955, -0.39176276326179504, -0.0697612315416336, 0.4879503846168518, -0.6106366515159607, -0.467766672372818, -0.9772278070449829, -0.8010947704315186, -0.6224466562271118, 1.2764226198196411, 1.8465325832366943, 0.2594199478626251, -1.5075868368148804, 1.4296585321426392, -0.14834746718406677, -0.3799789547920227, -1.8313440084457397, -0.7223100662231445, 0.8153620362281799], [-0.799814760684967, 1.163859248161316, -1.1409772634506226, 0.04420960694551468, 0.5971572399139404, 0.1964721381664276, -0.34500449895858765, 0.060605648905038834, 1.2884902954101562, -1.3639016151428223, -0.2841297686100006, -0.6905511021614075, -1.1765817403793335, 0.05524066835641861, 1.4369263648986816, 1.27094304561615, 0.5304642915725708, -0.6284843683242798, 1.953637719154358, -0.22740967571735382, 0.869345486164093, 0.5884445905685425, -0.8379330039024353, 0.6374285817146301, -0.5055539608001709, 0.13700613379478455, 0.8273974061012268, 1.525519609451294, 0.032441843301057816, 0.6993154287338257, 1.3211649656295776, 0.9165491461753845, 1.5486587285995483, 0.7159079313278198, -1.1839793920516968, -1.9755280017852783, -0.3765788972377777, 1.4567947387695312, 0.541283369064331, -1.3814482688903809], [-1.820586919784546, -0.1726553738117218, -0.6590491533279419, 0.2626701593399048, -0.11082124710083008, -1.06021249294281, -0.11196491867303848, -0.370302677154541, 0.21685466170310974, 1.0531800985336304, -0.1520209014415741, -0.8449867963790894, 0.005216435994952917, 1.3670103549957275, -2.299330234527588, 0.21851196885108948, -0.28170230984687805, 0.6282097697257996, 0.5535614490509033, -0.3060266077518463, 1.5434811115264893, 0.4419652819633484, 1.0687150955200195, -0.6062241792678833, 0.6957404613494873, -0.05045849457383156, 0.38014936447143555, 2.0415432453155518, -0.7617461085319519, 0.7389541268348694, -0.7970295548439026, -1.3078895807266235, -0.09456693381071091, -0.31239667534828186, -0.022282907739281654, 1.2020351886749268, -1.1363052129745483, 0.8558604121208191, 1.0735597610473633, 2.3395211696624756], [-0.7004791498184204, 0.26346850395202637, 0.19073359668254852, 0.5122659206390381, 0.3301597237586975, 0.9208044409751892, -0.9731014966964722, -0.5522767305374146, 0.5432155132293701, 0.05912709981203079, 1.029185175895691, 0.9246838092803955, 0.36186185479164124, -0.1905951201915741, 0.657804012298584, -0.2724246084690094, 0.24175241589546204, -0.23431779444217682, -0.7600876688957214, 0.8812006711959839, -0.24621890485286713, 0.9409691691398621, -0.9946056008338928, 0.9169859886169434, 0.5733643174171448, -0.7120438814163208, 0.4606781601905823, 0.45055529475212097, -0.5927125215530396, 0.4366544783115387, 0.16794408857822418, 0.4412228763103485, 1.3337172269821167, -0.48037436604499817, -1.2877459526062012, -0.16231919825077057, -1.0513845682144165, 0.33104074001312256, 0.13496065139770508, 0.5185027718544006], [0.33159396052360535, -0.15617212653160095, 0.1665239930152893, -1.4647245407104492, -0.8076308965682983, 1.7554501295089722, 1.0902961492538452, 0.38484665751457214, -0.27664557099342346, 0.8230011463165283, -0.2033318728208542, 0.46225640177726746, -0.31333306431770325, -0.06285331398248672, -0.8425437808036804, 0.6209681630134583, 1.5271464586257935, -0.6686620116233826, 2.0924205780029297, 0.6231271028518677, -0.1107637882232666, 0.49629828333854675, 0.6635889410972595, 1.6957218647003174, 0.7331247925758362, 0.24748601019382477, -0.45465463399887085, 0.45298856496810913, -0.07910960167646408, 0.13397230207920074, 0.32740578055381775, -1.0481760501861572, 0.560029149055481, -0.8981238603591919, 1.4662812948226929, -0.009271468035876751, -1.4561734199523926, -0.2810526192188263, 0.11947450041770935, 0.2687336206436157], [1.4364408254623413, -0.8920921087265015, -0.5305579900741577, 0.10979323089122772, -1.9531484842300415, 0.4408511817455292, -0.22116179764270782, -1.830178141593933, -0.2851441204547882, 1.4914238452911377, 0.0876380205154419, -0.38418182730674744, 0.11345303803682327, -1.0700838565826416, -0.2254798263311386, 0.46026891469955444, 0.3772529363632202, -0.8225371837615967, 0.982124388217926, 1.315507411956787, -1.30784010887146, -1.96296226978302, -1.2131083011627197, 0.6332516670227051, 0.23995335400104523, 2.1528899669647217, 1.0320854187011719, 0.4616077244281769, -0.2867545187473297, -1.3015615940093994, 2.120941638946533, -0.9582741856575012, 1.3137253522872925, -0.7696626782417297, 0.7222545146942139, -0.00042694382136687636, -0.08139878511428833, 0.8511884212493896, -0.02654767408967018, -0.28808358311653137], [0.8857593536376953, -1.0136834383010864, 0.367508202791214, -0.3097477853298187, 0.5154313445091248, -1.5083348751068115, -2.099109649658203, -1.5927364826202393, 0.7253448367118835, 0.0024040888529270887, -0.5727354884147644, 1.1659839153289795, 0.6555507183074951, 0.7551299929618835, -0.8047441840171814, 0.32965126633644104, 0.34274107217788696, 1.2491581439971924, 1.5996583700180054, -1.9714769124984741, 0.5011268258094788, -0.6199366450309753, 0.24048788845539093, 0.39808347821235657, -0.5423189401626587, -1.520763635635376, 0.29135411977767944, -1.0042293071746826, -0.8606497049331665, 0.38341569900512695, 1.270268440246582, -0.31606757640838623, -1.2203457355499268, -0.44296687841415405, -0.7808681726455688, 0.8338783979415894, -0.7188910841941833, 1.3399137258529663, 0.08268988132476807, -1.9859284162521362], [-0.3112553656101227, 1.3917429447174072, 2.6503050327301025, 0.5278177261352539, -0.7100653648376465, 0.18239879608154297, 2.000577449798584, 0.8127548098564148, 1.8489444255828857, -0.11999522149562836, -0.6632816195487976, -0.38187849521636963, 0.7742943167686462, 0.20812207460403442, 1.9567469358444214, -1.1010257005691528, -0.36942368745803833, -0.11181650310754776, 0.7397664785385132, 2.0290918350219727, -1.3838005065917969, 1.8940620422363281, -0.42783793807029724, -0.2379358559846878, 0.06950478255748749, 1.848799228668213, -0.59434574842453, 0.5535020232200623, -0.7138540744781494, 0.5157583951950073, 1.2431672811508179, -1.006608486175537, 0.8822296857833862, 0.6084553599357605, -1.0138784646987915, 1.1555159091949463, -1.0341848134994507, 1.2184945344924927, 0.22375856339931488, -1.2159343957901], [-0.7977853417396545, 1.1377620697021484, -3.281633138656616, -1.2327189445495605, -0.7433410286903381, -0.5184493064880371, -0.6039844751358032, 0.8092578053474426, -0.0886828824877739, -0.9467207193374634, 0.1723501980304718, 1.0784648656845093, 1.4969990253448486, 0.0699150562286377, -0.7812180519104004, -0.8974624276161194, 0.1359240561723709, -0.4736943542957306, 0.850246250629425, -1.3756277561187744, -3.0197110176086426, -0.015016568824648857, -0.7841138243675232, 0.5208748579025269, -0.6810708045959473, 0.2742469310760498, 1.203344702720642, 1.5694586038589478, 0.8998990058898926, -0.639664351940155, -0.2697312533855438, -0.0025852045509964228, -2.1874308586120605, -0.8119854927062988, -0.5235704779624939, 0.35351112484931946, -0.12018690258264542, 1.5648037195205688, 0.7886812686920166, -0.5493055582046509], [0.9138131737709045, -1.1602046489715576, -0.7660880088806152, -0.15733122825622559, 1.263839602470398, -0.4028162658214569, 1.279923439025879, 1.1864551305770874, 1.6470332145690918, -1.2269916534423828, 1.0191097259521484, -0.46140575408935547, -1.8078428506851196, -0.4988616108894348, 0.4454076886177063, -0.013319503515958786, -1.05965256690979, 1.565903902053833, -0.2443884015083313, -0.4805445075035095, -0.6192026734352112, 0.08477779477834702, -0.6105842590332031, -0.1121767908334732, -1.4012682437896729, -1.8148857355117798, 1.2933058738708496, 2.7566323280334473, -0.6883172988891602, -0.435577392578125, -0.15576112270355225, -1.3452945947647095, -0.3941754400730133, -0.5273305177688599, -0.6631352305412292, -0.29998457431793213, 0.9843283295631409, -1.1791610717773438, -0.3248996436595917, 0.6646759510040283], [0.6636826395988464, 0.5608839392662048, 0.6319074630737305, 1.0942542552947998, -0.06503576785326004, 1.1657848358154297, -0.9895414710044861, 1.043105125427246, 0.8254257440567017, 0.7444784641265869, -0.7763607501983643, -0.8335952162742615, -0.8893611431121826, 0.11136544495820999, -1.8299707174301147, -1.0332683324813843, -0.9046638607978821, -1.3134156465530396, 0.46871334314346313, -0.9053398966789246, 0.05599594488739967, 0.9587734341621399, 0.6383208632469177, -0.12067235261201859, -0.6234161853790283, -0.2870631814002991, 1.6759051084518433, 1.8953856229782104, -0.3903234601020813, -0.7741686105728149, -0.8528608083724976, 0.03217419236898422, -0.24157319962978363, -0.09685439616441727, -0.8002324104309082, -0.7857639193534851, 1.1295539140701294, -1.0672876834869385, -0.4806739091873169, 0.34012818336486816], [0.8713388442993164, 0.645469605922699, 0.34918737411499023, -0.7092419266700745, -0.4025407135486603, -0.5967934131622314, 1.056013584136963, 0.19699722528457642, -0.06476394832134247, -0.10333862155675888, -0.501737117767334, 0.2687591016292572, -0.8678420186042786, -0.20108628273010254, 0.7896770238876343, -0.0627320259809494, 0.8980973958969116, 1.4031635522842407, -0.06133841350674629, 1.4327572584152222, -0.5606727004051208, 0.36339086294174194, 1.0340349674224854, -1.5710216760635376, -0.1912466287612915, 0.910614013671875, -0.5779625177383423, -0.026186374947428703, 0.39116063714027405, 0.15785999596118927, 0.4486827850341797, -0.006895471829921007, -0.18836896121501923, -1.2964565753936768, -0.2263556718826294, 0.3529316782951355, 0.62334144115448, 1.6275006532669067, -0.6636310815811157, 1.3266571760177612], [-1.1809697151184082, -2.731828451156616, 0.3908974826335907, -0.7232217788696289, -0.10235759615898132, -0.28367510437965393, -0.1479233354330063, 0.18073630332946777, -0.5040984153747559, 0.16413448750972748, -0.34896165132522583, 0.6663795709609985, 0.22415947914123535, 1.0860942602157593, -0.17035841941833496, -0.141315296292305, 1.0668182373046875, -1.2675824165344238, 0.04021676257252693, 1.5397835969924927, 0.7391476035118103, -1.590538740158081, 0.0914679542183876, 1.2382652759552002, -0.5664597153663635, 0.7229092121124268, -0.3778811991214752, 1.4222090244293213, -0.33971068263053894, 0.13861803710460663, 1.2706971168518066, 1.185210943222046, 1.213014006614685, 0.3075571358203888, -2.9853975772857666, 0.28121599555015564, 0.7352863550186157, 0.15559065341949463, 0.18139687180519104, 1.895173192024231], [-0.8560278415679932, -0.3709923326969147, 1.030299186706543, 1.3176058530807495, 0.09317243099212646, -1.0760087966918945, -0.04698045551776886, -1.0073859691619873, 1.5261106491088867, -0.24346545338630676, -1.2826964855194092, 0.33866798877716064, -0.6814737319946289, 1.0132250785827637, -0.0048241158947348595, 0.9539010524749756, 2.552537679672241, 0.17845487594604492, -0.6461747288703918, 2.759822368621826, -0.3483674228191376, -0.6246792674064636, 1.8589110374450684, -0.6057645678520203, 2.170285224914551, -0.33748695254325867, -0.4491848945617676, -0.4116503894329071, -0.8608397245407104, 0.3162785768508911, 1.1954419612884521, 0.41691407561302185, -0.40837424993515015, 1.0172810554504395, 0.4765099883079529, 1.5809770822525024, 0.48875296115875244, -1.9758645296096802, 0.38235655426979065, -0.10086466372013092], [0.34698474407196045, 0.9572117328643799, -0.05824147164821625, -0.9767197370529175, -1.5518535375595093, -1.3033866882324219, 1.7090684175491333, 0.7360993027687073, -0.2762198746204376, -0.09766003489494324, 1.721582055091858, -0.12535950541496277, -0.8515220284461975, 0.6441591382026672, 0.5429179668426514, 0.03154579922556877, -0.209792822599411, -1.2614191770553589, 0.4029979407787323, -0.6230043172836304, -1.0036029815673828, -0.13339170813560486, 0.3520418405532837, 0.12762035429477692, 0.7283642292022705, -0.9807743430137634, -0.06007760390639305, -1.2115528583526611, 1.4894107580184937, -2.1261050701141357, 0.5520702004432678, -1.0571238994598389, -0.6594226956367493, -0.7381922006607056, 0.35330629348754883, -0.20960773527622223, 1.4554473161697388, 0.35935693979263306, 1.916040301322937, 2.2960965633392334], [1.2075567245483398, -1.2106070518493652, 0.9040331840515137, -1.243243932723999, -1.1136471033096313, -1.4506831169128418, -0.017145315185189247, 1.756929874420166, 1.227952241897583, -1.4562256336212158, 0.44500550627708435, 1.3031878471374512, 1.2044169902801514, 0.5442047119140625, -0.1893942505121231, 0.47766244411468506, -0.29136449098587036, -0.01864420436322689, -0.44690388441085815, -0.29613348841667175, -0.8860977292060852, 0.9695389270782471, 0.3655985891819, -0.019609851762652397, 0.007614381145685911, -0.2817244529724121, -0.3496650755405426, 0.8211343288421631, 1.528118371963501, -0.6703581809997559, 0.2259054034948349, -0.5021086931228638, -1.1389451026916504, 0.018320759758353233, 0.5677133798599243, -2.956711769104004, 0.47051823139190674, 0.5920559167861938, 0.8701680898666382, -1.5928393602371216], [1.2215185165405273, 1.0632308721542358, -1.2994643449783325, 0.769931972026825, -1.0465093851089478, 1.4533106088638306, 0.03861600160598755, -0.986984372138977, -0.010651258751749992, -1.145020842552185, 0.07874967902898788, -0.3075241148471832, 0.2247587889432907, -1.1047829389572144, -1.3252443075180054, -0.5421178340911865, 1.1388081312179565, 1.737901210784912, -0.57597416639328, 2.2473573684692383, 0.5257665514945984, 0.7779200673103333, 0.6729477047920227, -1.2580987215042114, -0.9835947751998901, -0.20042946934700012, -1.2994378805160522, 1.0790760517120361, 0.6185248494148254, 0.014714802615344524, 1.4314454793930054, 0.8728004097938538, 0.06979869306087494, 0.788586437702179, -1.300355315208435, -1.3289599418640137, -0.5082409977912903, -0.3239288926124573, 0.20592200756072998, -0.3515532314777374], [-0.033919379115104675, -0.09650641679763794, -0.3016476035118103, -0.7669374346733093, 0.5820090174674988, -0.6189934015274048, -0.721894383430481, -0.25708210468292236, 0.5424323678016663, 2.512272834777832, 0.06798025220632553, 1.0129647254943848, -0.46628955006599426, -0.4856550693511963, 0.11580763012170792, 0.9734477400779724, -1.1492210626602173, 0.6874805092811584, 1.2001816034317017, 0.9818174839019775, 0.23340408504009247, 0.8951180577278137, -0.5119217038154602, -0.9662236571311951, -0.056004080921411514, 1.3608558177947998, 0.7735506892204285, 2.2313151359558105, -0.9779200553894043, -0.6299653053283691, -0.18895244598388672, -0.21863006055355072, -0.6523273587226868, 0.558108925819397, -1.5621906518936157, 1.0549004077911377, -0.17506133019924164, -0.835115909576416, 0.2130480706691742, -0.6437910199165344], [-2.2564890384674072, -0.8215656876564026, -2.552727460861206, -2.0881826877593994, 0.38770946860313416, 0.06254492700099945, 0.49117374420166016, 0.22611801326274872, 0.3623215854167938, -0.2887020707130432, 1.578834891319275, 1.0258396863937378, -0.274845153093338, 0.7261521220207214, 1.1469476222991943, -0.33147355914115906, -1.3250977993011475, 0.759009063243866, -0.7588097453117371, 0.9224872589111328, 1.101952314376831, 1.0887274742126465, -1.0934321880340576, 1.4303185939788818, -1.321498990058899, 0.15404288470745087, -1.86152982711792, -0.28041204810142517, -0.19134639203548431, 2.4472670555114746, -0.5443918704986572, -1.8306845426559448, -0.5318371057510376, -1.2967196702957153, -0.489650696516037, -0.6893675923347473, 1.3731619119644165, 2.04170298576355, -0.4473606050014496, -1.4836750030517578], [0.6623387932777405, -0.14593176543712616, 0.6613163352012634, 0.40292030572891235, -0.004644783679395914, 1.2064714431762695, -0.7105910181999207, -0.4231988191604614, -0.8710466623306274, -1.5252920389175415, 0.9880797266960144, 0.46158042550086975, -0.02598092146217823, 0.10943087190389633, -0.5157417058944702, -0.7805741429328918, 1.9148850440979004, 1.0458393096923828, -0.042263783514499664, 0.3593864142894745, 1.0523426532745361, -0.2611640691757202, -0.1411028504371643, 1.462817668914795, 1.403457760810852, 0.5643385052680969, -1.0271632671356201, -0.28167450428009033, 2.487900495529175, 0.05064108967781067, 2.044726848602295, -0.8601669073104858, -1.912627100944519, -0.21981710195541382, -3.314885139465332, -0.018714603036642075, 0.7206798195838928, 0.47211259603500366, 0.20963656902313232, 0.2628202438354492], [-1.7080798149108887, -0.7582089304924011, -0.05849325656890869, 0.28205087780952454, 0.783857524394989, 1.2776662111282349, -0.2906264066696167, 1.2140133380889893, -0.009745562449097633, -0.7568237781524658, 1.4998112916946411, -1.0844696760177612, 1.3425898551940918, -1.0396926403045654, -0.27071428298950195, 0.7176339030265808, 1.9698677062988281, 0.3801044523715973, -0.0905301421880722, -0.9454714059829712, 0.07760882377624512, 0.8625431656837463, -0.9519509673118591, 1.3362300395965576, 0.0640333816409111, -0.23743821680545807, -0.09238570183515549, 0.7710942625999451, 2.0055992603302, 1.7661309242248535, 0.7557488083839417, -0.23137158155441284, 0.003387338947504759, -0.22999803721904755, 2.0623326301574707, -0.2894761264324188, -0.5825497508049011, 0.3438209295272827, -0.43524694442749023, 1.9716423749923706], [-1.136314034461975, -0.5152036547660828, -0.9743409752845764, 0.5535959005355835, -0.18726949393749237, -0.4576895534992218, -0.11022467166185379, 0.37293267250061035, -0.4361729323863983, 0.11542307585477829, -0.08890689164400101, 1.5533068180084229, 1.2255821228027344, -0.35667234659194946, 1.1821645498275757, 1.4923417568206787, 0.7930570244789124, 0.5815364718437195, -0.5968374609947205, 0.0171566940844059, 0.012875713407993317, 1.7145134210586548, 0.9751288294792175, -0.11979831010103226, 1.126692295074463, -0.2613053321838379, 0.2766283452510834, 0.40878671407699585, -0.15193958580493927, -0.9939437508583069, -0.08143401890993118, -0.9281817674636841, -0.0041145640425384045, -0.15370690822601318, 0.7794675230979919, -1.7868516445159912, -1.1877212524414062, 1.1872758865356445, 0.1385214924812317, 0.39994680881500244], [0.953030526638031, -0.17965790629386902, -0.08045920729637146, -0.9252704381942749, -0.4356246292591095, 1.9095014333724976, 0.6343032121658325, -0.589244544506073, -0.046793676912784576, -0.5685868859291077, 1.2028237581253052, 1.3268680572509766, 0.797722578048706, -1.288387656211853, 1.1275601387023926, -0.20998050272464752, 0.030088424682617188, 0.0129550127312541, -1.7580647468566895, -2.5356698036193848, -1.0818232297897339, 1.397620439529419, -0.30135318636894226, 0.5961204767227173, -0.7837403416633606, 1.218896746635437, 0.2784535884857178, -1.3135709762573242, -0.5489895939826965, 0.4565364420413971, 1.3542532920837402, 2.756448268890381, -1.3936060667037964, -1.0979396104812622, -0.6493796110153198, 0.8575683236122131, 0.9026766419410706, -0.5176511406898499, -0.5609318017959595, -1.746827244758606], [0.12926626205444336, -0.7400323152542114, 0.15459582209587097, 0.01609024964272976, -0.9695549607276917, 1.0052592754364014, 0.7064287662506104, 1.2593364715576172, -0.45735570788383484, 0.9147927165031433, 1.1491073369979858, -0.8111613988876343, 1.059537410736084, -0.10224483162164688, -1.5905946493148804, -0.20858757197856903, -0.043634772300720215, 1.038726806640625, 0.6671385169029236, -1.0925490856170654, 0.0043007927015423775, -0.6020197868347168, -0.9175573587417603, 1.4240283966064453, -0.6115344166755676, -0.1684069186449051, 0.19383719563484192, 0.06549141556024551, 0.24742117524147034, 0.4068923592567444, -0.561307966709137, -0.06109005957841873, -0.2740672826766968, 0.022829929366707802, 0.33568888902664185, -0.8546485304832458, -0.9586998224258423, 2.23097825050354, 0.5881773829460144, -1.2897322177886963], [0.11423858255147934, 0.4803241789340973, 1.0723661184310913, -0.49101722240448, 0.885464072227478, -1.6216678619384766, -0.4715537428855896, -0.7799711227416992, -1.6942425966262817, -0.400077760219574, -0.23829002678394318, -0.8518046736717224, 0.5536099672317505, -0.033865831792354584, 0.4158446192741394, 0.9820785522460938, 0.4588903486728668, -0.540217399597168, 0.2843109965324402, -1.2546613216400146, -0.3677556812763214, -1.0879411697387695, 0.11255309730768204, 0.19505172967910767, 0.015919681638479233, -0.3337337374687195, -1.9200310707092285, -1.0871888399124146, -0.9688060283660889, -0.9345642924308777, 0.023429417982697487, -0.33336421847343445, 0.25053319334983826, -0.48544371128082275, 0.45461979508399963, 0.4231676757335663, -1.581261396408081, -1.1755335330963135, 1.5021532773971558, -1.5085017681121826], [-0.971077561378479, 1.3565400838851929, -1.729455828666687, -0.26205193996429443, 0.3715842664241791, -0.5630987882614136, 0.3300066888332367, -0.9679756164550781, -0.4352734684944153, -2.034992218017578, -0.11871238052845001, 0.2939983308315277, -0.8252646327018738, 0.08715645968914032, -1.3437048196792603, 1.5303212404251099, 1.056151032447815, -2.2544808387756348, 2.2186086177825928, 0.3634217083454132, 1.2225067615509033, 0.35755833983421326, 0.15223605930805206, 0.8760189414024353, -0.10281022638082504, -0.14114217460155487, 0.5171672701835632, -2.0611939430236816, 0.46411651372909546, -1.3929104804992676, 0.12405914813280106, -0.5622842907905579, -0.6961389183998108, 0.591191828250885, 0.8729807138442993, -1.1850495338439941, 0.12018033862113953, -0.36839979887008667, -1.263388752937317, -0.8810743093490601], [-0.036135632544755936, 1.4953656196594238, -1.393638014793396, 1.240722894668579, 0.39436519145965576, -0.9947909712791443, 0.5122275352478027, 0.5793821215629578, 1.1685235500335693, -0.3035591244697571, 3.4104456901550293, -1.2656500339508057, -1.1537339687347412, -0.010921078734099865, -1.0422247648239136, -0.14383411407470703, -1.6208314895629883, -0.46171480417251587, -0.40766778588294983, -0.1999032199382782, -0.7656275629997253, -0.28524214029312134, -0.43178996443748474, -0.212417334318161, 1.739723801612854, -2.098700523376465, 0.36520907282829285, 0.7500342130661011, -1.1343085765838623, -0.2815299332141876, -1.3140865564346313, -0.5346788167953491, 0.5406034588813782, 0.9961074590682983, 0.05274577438831329, 0.2033849060535431, -0.08469349145889282, -1.8190358877182007, 0.8469314575195312, -1.4504872560501099], [-0.8461325168609619, -0.49407798051834106, -0.37755119800567627, -0.6638143062591553, -1.0000054836273193, 0.6108126044273376, 0.27983981370925903, 1.6269487142562866, -0.542553722858429, -1.5914591550827026, -1.073011040687561, 1.4637088775634766, -0.3798186480998993, 0.35173794627189636, 0.7606516480445862, 0.5968771576881409, 0.28022241592407227, -0.029750213027000427, 0.4184236228466034, 0.5658871531486511, -0.9415152668952942, 0.7500019669532776, 0.36252403259277344, 0.800112783908844, -1.391992211341858, -0.763382077217102, -1.727957010269165, -0.9528340101242065, 0.658041775226593, 1.3145993947982788, -1.4372589588165283, -0.12168969959020615, -0.35473141074180603, -0.27532562613487244, 0.4891667664051056, -0.41606244444847107, -1.2094721794128418, 0.3119347095489502, 0.4105819761753082, 0.5759166479110718], [1.2351020574569702, -0.3985377550125122, -1.2257483005523682, 0.9856733679771423, -0.7287871241569519, 0.24042809009552002, -0.9684155583381653, 0.13813096284866333, 1.413332223892212, 0.15451686084270477, 0.36106202006340027, -0.35060107707977295, 0.38023093342781067, -0.33156129717826843, -1.1203429698944092, 1.676445722579956, -0.3951934576034546, -0.6562418341636658, -1.0318150520324707, -0.27774950861930847, -1.6682755947113037, 0.3435474634170532, 1.487326741218567, 0.1833784580230713, 0.9149009585380554, 1.017134189605713, 1.2308562994003296, -0.2436893731355667, -0.01676351949572563, -1.3761610984802246, 1.9438341856002808, 0.31758153438568115, -0.7806176543235779, 0.7400519251823425, 0.7254684567451477, -0.25024908781051636, 0.1619747281074524, -0.7582290172576904, 0.7586326599121094, -1.270078182220459], [1.58896803855896, -0.5857967138290405, -1.0210477113723755, 0.3025681972503662, -0.7147098779678345, 0.4759069085121155, -0.7025877833366394, 0.5790582299232483, 0.49800488352775574, -1.482056975364685, 1.5006600618362427, -0.8285547494888306, -0.10513115674257278, 0.4735822081565857, 0.6930806040763855, 0.3315306305885315, 0.7929640412330627, 0.8515488505363464, 1.2578492164611816, 1.2156425714492798, 0.8632575869560242, -1.2349621057510376, 0.590944230556488, 1.7316867113113403, -0.9315787553787231, 0.35908225178718567, 0.09741153568029404, 0.37271320819854736, -0.25257226824760437, 0.14542528986930847, 0.27073749899864197, -0.547989010810852, -0.1228659600019455, 0.1934112161397934, 0.7078967094421387, -0.6387565732002258, 1.0308789014816284, 0.5983256697654724, -0.8641290068626404, 1.0831907987594604], [-0.28332722187042236, 0.4256453812122345, -0.3737300634384155, 0.3045607805252075, 0.6542848944664001, 0.5640352964401245, 0.7648643255233765, 1.2931060791015625, -0.5428321361541748, 0.06260732561349869, -1.153592824935913, -0.4094955623149872, 1.0340152978897095, 0.7465207576751709, 0.25788208842277527, 0.5721014738082886, -0.04860920086503029, 0.18838663399219513, 0.13500192761421204, -0.9489450454711914, -0.8627253770828247, -1.3298884630203247, 1.3527952432632446, 0.9377568364143372, 0.011038322001695633, -0.11226899921894073, 1.2552266120910645, -1.8896424770355225, -0.8957633376121521, -1.2501472234725952, -1.529636263847351, -0.2658395767211914, 1.2576137781143188, -0.4173993468284607, -0.4558714032173157, 0.3590494394302368, 2.0302953720092773, 1.2576603889465332, 0.442234605550766, 0.7292651534080505], [-0.4026936888694763, -1.337913155555725, -0.03641299530863762, 0.028612257912755013, -0.018633216619491577, 0.40142306685447693, 2.161975145339966, 3.395098924636841, -1.4535773992538452, 1.122525930404663, 0.7273936867713928, 0.7671079635620117, 0.43467405438423157, -0.9945307374000549, 1.5539917945861816, 1.8615381717681885, 0.1579829752445221, -0.9653528332710266, -0.28676003217697144, -0.3936012089252472, 0.15612836182117462, 0.739651620388031, -2.0462207794189453, -0.9448510408401489, 1.2321670055389404, 1.4425082206726074, 0.8053591251373291, 0.028400545939803123, -1.073912501335144, 1.015071153640747, -1.6062705516815186, -0.33715546131134033, -0.8597851991653442, 2.444882869720459, -1.739330768585205, -1.976839542388916, 0.26769715547561646, 0.5731582641601562, -0.05050298199057579, -0.4673357903957367], [-0.6100842356681824, 0.5154479742050171, 1.6324275732040405, 0.6808655858039856, -0.0729425922036171, 0.13458193838596344, -0.4167650043964386, -0.446776807308197, 0.05550144240260124, 0.09030772000551224, -0.7949666380882263, -2.5058727264404297, 0.5076711773872375, -0.22736279666423798, -0.40742915868759155, -0.11935915797948837, -0.1603023111820221, -0.1753549426794052, -0.18616293370723724, 0.5192446708679199, -0.2210867702960968, -0.8131019473075867, 0.6482542753219604, -0.22985747456550598, -0.11281737685203552, 1.1220378875732422, 0.33721616864204407, -0.3199153244495392, -0.019523337483406067, -0.3100963234901428, 0.6663471460342407, 0.08016298711299896, -0.38191327452659607, 0.6437734365463257, -0.34616345167160034, -1.6880857944488525, -0.963721752166748, -0.17978820204734802, 1.5918689966201782, 0.4298720359802246], [-0.8118375539779663, -1.5134316682815552, -0.666678786277771, -1.173456072807312, 1.0464146137237549, -0.185796856880188, 0.35083696246147156, 0.8522812724113464, 1.2650401592254639, 0.009061095304787159, -0.09410551190376282, -0.5001296997070312, -0.8636944890022278, -0.8978562951087952, -0.4670361876487732, -1.4013270139694214, 0.02328425832092762, -0.1637210249900818, 0.15926077961921692, 0.3521035611629486, 1.5739110708236694, 0.4489521086215973, 1.1941015720367432, 0.14080369472503662, -1.480357050895691, -0.4967900514602661, -0.1806338131427765, -0.39312317967414856, -0.052234601229429245, 0.21702159941196442, 1.5246206521987915, -0.7744303941726685, 1.5364391803741455, 0.6041550636291504, -0.9812207221984863, 0.3750244975090027, -0.2838728427886963, -1.1664769649505615, 0.22878225147724152, -0.4544174075126648], [-0.6325739622116089, 0.8367518186569214, 1.0249793529510498, 0.34104493260383606, -0.7101792097091675, 0.43682146072387695, 1.5316619873046875, 0.09623782336711884, -0.5942764282226562, 0.03673381730914116, -0.4635452628135681, -0.473453551530838, -0.8835323452949524, 0.0980047732591629, 0.5093830823898315, -1.9350395202636719, 0.8363552689552307, 1.2967774868011475, -0.7980175018310547, -0.8089671730995178, -1.5006996393203735, 0.9486944079399109, -0.12768249213695526, -0.9646196961402893, -1.7833125591278076, 0.7967996001243591, -0.6194765567779541, 1.0414135456085205, 0.4349729120731354, 0.7251282334327698, -1.1643102169036865, -1.0183022022247314, 0.02303028292953968, 0.6140012741088867, 0.13858763873577118, 0.6766043305397034, 1.904781699180603, 0.9846130013465881, 0.23714962601661682, 1.5797197818756104], [-1.3452023267745972, 1.3066363334655762, 1.2526150941848755, 0.40704962611198425, -0.343069463968277, 1.5083032846450806, 0.138133242726326, -0.5880265235900879, 0.12611453235149384, -0.1327018141746521, -0.8721880912780762, 0.3782508373260498, -0.1628006398677826, 0.40499159693717957, -0.9643969535827637, 1.100513219833374, -0.08003958314657211, 0.5539774894714355, -1.433970332145691, 0.7263133525848389, -0.5691168308258057, 0.27703946828842163, 0.6540094614028931, -0.22508208453655243, 0.969180703163147, -1.2604269981384277, -1.4650212526321411, -0.2975383400917053, -0.3607542812824249, 0.6348240971565247, -1.8747223615646362, -0.22282883524894714, -0.7073485851287842, -0.7378658652305603, 0.41920220851898193, -2.444979190826416, -0.5986527800559998, -0.10196994245052338, -1.1179426908493042, -0.4030732810497284], [0.3524097800254822, -1.6573258638381958, -0.5396472215652466, -0.2310064285993576, 0.79942786693573, -1.0694938898086548, 1.401146411895752, -0.29402387142181396, 0.36136168241500854, 2.319044828414917, 2.360978603363037, 1.0401092767715454, -0.8547921776771545, -0.6581243276596069, -1.2361674308776855, -0.342686265707016, 0.7660204172134399, 1.081433653831482, 0.5647810101509094, 0.2225555032491684, 0.13324201107025146, 0.8093292117118835, 0.11473517119884491, 0.08950159698724747, -0.4593575596809387, 1.942012071609497, 2.2567243576049805, -1.249788522720337, 0.5552776455879211, 1.448818564414978, -0.34893935918807983, -0.5962889790534973, -2.161259889602661, -1.0875155925750732, -0.3503977060317993, 0.8951502442359924, 0.7255973219871521, 0.6964253783226013, -0.17785456776618958, 0.33416303992271423], [0.5156971216201782, 1.4756178855895996, 0.8347569704055786, -0.9583831429481506, -0.3297063708305359, 0.5407626628875732, -1.4457354545593262, 0.5956100821495056, -0.30796512961387634, -1.133905053138733, 0.848972499370575, -0.12339329719543457, 0.1416548192501068, -0.17484357953071594, -0.624944269657135, 1.4106730222702026, -0.4483332335948944, 0.43023839592933655, -0.799496054649353, -0.20755121111869812, -0.3620939552783966, 2.0131356716156006, 0.30252060294151306, -0.7395545244216919, 2.1060566902160645, 1.166792392730713, 2.5925023555755615, 2.031203269958496, -1.9374680519104004, -0.2081356644630432, 0.23182572424411774, -0.21508561074733734, -0.87873774766922, 0.09473900496959686, -0.2931974530220032, -2.754859209060669, 1.732218861579895, -0.08600800484418869, 0.46754083037376404, 1.1225792169570923], [0.5896970629692078, 0.4111121594905853, 1.1673591136932373, 0.25755375623703003, 1.807461142539978, 0.4191412329673767, 1.0332773923873901, -1.0543917417526245, 0.6184622049331665, 0.045516129583120346, 0.41632458567619324, -1.2310551404953003, 1.1132463216781616, 0.05958937853574753, 0.8428236842155457, -0.4898820221424103, 0.0031042161863297224, 0.2813796401023865, 0.8114736676216125, 0.13838063180446625, -0.24094431102275848, -0.17810632288455963, -0.6788736581802368, -0.5692956447601318, 0.18223130702972412, 0.6239932775497437, -0.7936842441558838, -0.10343924164772034, -0.08499690890312195, -1.0812489986419678, 2.4401073455810547, 0.6776307225227356, 0.16875430941581726, -0.36601290106773376, 0.10290762782096863, 0.9015209674835205, -0.534184992313385, 1.1708229780197144, -3.686204671859741, -1.3005386590957642], [-0.1122378334403038, 0.11236318945884705, -1.0103704929351807, -0.7122885584831238, -3.0573360919952393, -0.17228282988071442, 0.5765713453292847, -1.479998230934143, 0.45986759662628174, 0.06277825683355331, -0.4112086296081543, -1.4775443077087402, 1.3790738582611084, 0.046467769891023636, 2.033191204071045, 1.9665651321411133, 0.6265860795974731, 0.6037515997886658, -0.0010416783625259995, -1.3255058526992798, 0.7092178463935852, -0.13369807600975037, 0.6932709217071533, -0.1650608777999878, 0.40302419662475586, 0.5736246109008789, -0.6051661372184753, 0.47891491651535034, -0.3222021162509918, 0.38962727785110474, 0.4097439646720886, 0.42444926500320435, 0.501205325126648, -1.1058903932571411, 1.2222800254821777, 0.06283420324325562, -1.533476710319519, -0.9301376342773438, -0.17249596118927002, 0.30839213728904724], [-0.6998386383056641, 1.5626757144927979, 1.4981071949005127, -1.455511450767517, 0.3489095866680145, 0.6226486563682556, -0.7540993690490723, 0.508187472820282, 0.5036489367485046, 0.7276836633682251, -0.7009310722351074, 1.6675798892974854, 0.508336067199707, 0.13996034860610962, -0.1857323944568634, -1.8977330923080444, -3.4089083671569824, 0.6430517435073853, 1.712947130203247, 0.5046769976615906, -0.050297655165195465, -0.49215394258499146, 0.7338660359382629, -0.19931454956531525, -0.8680028915405273, 0.14385126531124115, -1.0762823820114136, -0.605635941028595, -0.4720231592655182, 0.24951203167438507, -1.2514896392822266, 0.7584063410758972, -0.14185833930969238, -0.6522873640060425, -0.772855281829834, -0.3256005644798279, -0.6587928533554077, -0.8485295176506042, 0.8125313520431519, -0.8487342596054077], [1.6523287296295166, 1.7127299308776855, -1.3771940469741821, -1.5617812871932983, 0.046012185513973236, 0.6696009039878845, 0.2818962335586548, -0.8018194437026978, 0.5262757539749146, -0.5195896029472351, 0.8060449361801147, -0.3018149137496948, 0.5184811353683472, 0.9798445701599121, 1.1553958654403687, -0.8395377397537231, -1.0924720764160156, 0.9653003811836243, 1.4231212139129639, -0.5551743507385254, -1.0044891834259033, 0.28387945890426636, -0.7929804921150208, 1.3642668724060059, 0.898149311542511, -0.5747445821762085, -0.6145170331001282, -1.0740149021148682, -1.518173336982727, 0.2825870215892792, -0.4243585169315338, -0.5628570914268494, -1.3559199571609497, -0.9126288890838623, -0.029557054862380028, -0.6502228379249573, 1.6584221124649048, 0.1972631812095642, -0.49437272548675537, 1.3372935056686401], [0.2344025820493698, 0.1569693237543106, -0.6923845410346985, 2.8046579360961914, 0.5095237493515015, -1.263667106628418, 0.7167114615440369, 0.37158212065696716, 1.1470345258712769, -0.10521391779184341, -0.5584912896156311, -0.1289769858121872, -1.2336704730987549, -1.9383792877197266, 1.0133658647537231, -0.8003487586975098, 0.14839725196361542, -1.1885267496109009, -0.9393366575241089, -1.1443744897842407, -0.9682826399803162, 1.005731463432312, 0.10019876062870026, 0.1633526235818863, -1.8510711193084717, 0.9329095482826233, 0.16851906478405, 0.9165838360786438, -0.6186774969100952, -0.9020617604255676, -2.1298158168792725, 0.25933098793029785, 1.0001646280288696, -0.9329228401184082, 1.7249277830123901, 1.1723699569702148, -1.4391423463821411, 0.20355866849422455, -0.8647111654281616, 1.9915924072265625], [0.9811616539955139, 1.8499616384506226, -1.177392840385437, 0.855599045753479, 0.010846691206097603, -0.4871969223022461, -0.6301828622817993, -1.3607571125030518, -0.019527683034539223, 0.033366311341524124, -0.07112759351730347, -1.1628228425979614, 0.6013075113296509, -0.5621272921562195, 0.06544063985347748, 0.7404682040214539, -0.5762511491775513, -0.06682821363210678, 0.2991897761821747, -1.8449426889419556, -0.958461344242096, 0.6872912049293518, 0.8546551465988159, -1.3801438808441162, 0.6306700110435486, 1.495629072189331, 0.09774769097566605, -0.37914565205574036, 0.22956272959709167, -1.1480422019958496, 0.6811789274215698, -1.2759008407592773, -0.21753950417041779, -0.1626497060060501, 0.3236047029495239, -0.14861325919628143, 0.4716271162033081, -1.2366156578063965, 1.3547618389129639, -0.9516928195953369], [-0.3153483271598816, -0.7446422576904297, -0.5159471035003662, -0.6797641515731812, -0.536148190498352, 0.4896187484264374, -0.5148053765296936, -0.5343934297561646, 0.3574289083480835, -1.527068853378296, -0.05587848275899887, 0.7542461156845093, -0.5226165652275085, -0.8127370476722717, 0.12650851905345917, 0.6513293981552124, -0.1431867778301239, 0.6219210028648376, -0.09697698056697845, 1.2113922834396362, -0.0229620561003685, -1.0000147819519043, 0.15836535394191742, -0.6361621022224426, 0.6594358086585999, -0.4883670210838318, 3.2454123497009277, 0.29288509488105774, -0.5367766618728638, 2.9933931827545166, -0.9001519083976746, 1.898729681968689, -0.4458733797073364, 0.6876399517059326, -1.1031488180160522, 0.8890512585639954, 3.290767192840576, 0.41898733377456665, 1.402723789215088, -0.8332030773162842], [1.4237581491470337, -0.4922531545162201, -1.2798691987991333, -0.30255845189094543, 0.4129568338394165, -0.9637906551361084, 0.17462977766990662, -0.3270075023174286, 0.1856931895017624, -0.19405324757099152, 1.6474473476409912, -0.21198448538780212, 0.7826839685440063, 0.11821665614843369, -0.9752416610717773, 1.3363120555877686, -0.6896040439605713, -1.25908625125885, 0.8333819508552551, 0.20904091000556946, 0.039485007524490356, -0.500619113445282, -0.04876236990094185, 0.8489806652069092, -1.0941728353500366, 1.889455795288086, -0.6509674191474915, 1.2407493591308594, 0.4994329810142517, -1.2544752359390259, -1.4721788167953491, -0.6569540500640869, 0.43691006302833557, 0.00431806081905961, -0.4357719421386719, -1.001213788986206, 0.2405780553817749, 1.4103285074234009, 0.3216018080711365, 0.12965050339698792], [0.020211150869727135, 0.22546139359474182, 0.012171476148068905, 0.20072966814041138, -0.5082498788833618, -0.7053282856941223, 1.058650255203247, -0.020438794046640396, 0.05925479531288147, -0.8454716205596924, 1.6366382837295532, 0.796159029006958, -1.1269783973693848, -0.49434882402420044, -0.9187346696853638, -0.5742445588111877, -0.45044562220573425, 0.662511944770813, -0.26819053292274475, -0.5583784580230713, 0.2758772671222687, -1.8099080324172974, 0.6990365982055664, 0.4625207185745239, -0.8296591639518738, -0.38919487595558167, 1.3316895961761475, -0.7059519290924072, -0.19274812936782837, 0.23940011858940125, 1.6384000778198242, -0.8305036425590515, -0.19832740724086761, 0.906212568283081, 0.31896519660949707, -1.1030100584030151, 0.2688112258911133, -1.280278205871582, 0.5355203747749329, -0.07538073509931564], [-2.9423441886901855, -0.5685314536094666, 0.8782970309257507, 0.6102928519248962, -0.6533032059669495, -0.04779922962188721, -0.803299069404602, -1.7357248067855835, -0.8540704846382141, -0.984389066696167, -0.33352839946746826, 0.27863454818725586, 0.48364946246147156, -1.1303296089172363, -0.7318028211593628, -0.10688461363315582, 1.5393658876419067, 0.6135627627372742, -0.27313724160194397, 0.31020405888557434, 0.4056433141231537, 0.10577652603387833, -1.4133319854736328, -1.328117847442627, 0.663247287273407, 1.0825364589691162, 0.6216940879821777, -0.7914063334465027, 0.3157669007778168, 0.38056448101997375, -0.7755217552185059, 0.17571772634983063, 0.3641010820865631, 0.4065876603126526, 0.0633646622300148, -1.3122342824935913, -2.0681393146514893, -1.071253776550293, 0.5346123576164246, -0.30145132541656494], [0.147246316075325, 0.615004301071167, 0.8385180234909058, -0.11176422238349915, -0.8396916389465332, -0.675831139087677, -0.39759713411331177, -1.0419178009033203, -0.3419574797153473, 0.5746292471885681, 1.9683674573898315, 1.3110013008117676, 0.6596526503562927, 0.39933499693870544, 0.25534749031066895, -0.565919041633606, -0.5007452964782715, 0.7354506850242615, 0.3870128095149994, 1.2051639556884766, -1.9538750648498535, 0.6717400550842285, 0.1994204819202423, -0.8723500370979309, 1.3339755535125732, 0.6314470767974854, -1.2622638940811157, 0.6460468769073486, 0.7589123845100403, 0.28148114681243896, 0.5832698345184326, 1.7466002702713013, 0.1840357929468155, -1.1499499082565308, 0.000581720843911171, 0.46591895818710327, 0.8123331069946289, -2.0739991664886475, -0.9109074473381042, -0.9640762209892273], [1.9882093667984009, -0.777572751045227, 1.687538743019104, -1.9864270687103271, -0.38710692524909973, 0.48329925537109375, -0.15353256464004517, -0.927233874797821, -1.6841648817062378, -0.03717410936951637, -0.500442385673523, -1.1880918741226196, 1.2496423721313477, 1.19225013256073, 1.5232067108154297, 2.044466733932495, -1.7439091205596924, 1.3005841970443726, -0.9083487391471863, -0.2591392993927002, 0.7759844064712524, 0.5742743015289307, 1.301196813583374, -0.6327978372573853, 0.8179061412811279, 0.2105567455291748, 0.0957983061671257, -1.2063826322555542, -1.3147807121276855, -1.0796942710876465, -0.03513183817267418, -1.2191036939620972, 0.33433568477630615, 0.6515160799026489, 1.2775424718856812, 1.129546046257019, 0.0542176254093647, 0.5281587243080139, 0.6236277222633362, -1.9619141817092896], [-0.7191206812858582, 1.0640411376953125, -1.7281064987182617, -0.7643722295761108, 0.6182258725166321, 0.5701510310173035, -0.143244668841362, 0.18625149130821228, -1.1600921154022217, 0.7959992289543152, -1.1754493713378906, 0.23383304476737976, -0.3594169616699219, 0.4912625253200531, 0.06821408867835999, -0.545125424861908, 0.31471240520477295, 0.9908275604248047, -0.1201307624578476, -0.07259748131036758, -0.0035267428029328585, 0.7241688370704651, 0.4602595865726471, -1.3223093748092651, 1.219234824180603, 0.11165619641542435, 1.6132781505584717, 0.6160520911216736, -0.7171443700790405, -1.2088021039962769, -1.3827540874481201, -0.8276256322860718, -1.2593982219696045, 1.7637126445770264, -0.40195292234420776, -0.18571028113365173, 0.13751274347305298, -0.6912135481834412, 0.32737916707992554, -0.6652418375015259], [0.698053777217865, 0.4773905575275421, -1.2897326946258545, 0.18301987648010254, 0.25565779209136963, -0.5448591709136963, 1.1417983770370483, -0.20785169303417206, -0.8853996992111206, -0.27009817957878113, -1.652524471282959, -0.2076931893825531, 0.38814660906791687, -0.8604938983917236, 0.32402661442756653, 1.4459367990493774, -0.32713183760643005, 0.13028584420681, -1.6244686841964722, 1.9915969371795654, 0.19150106608867645, -1.533787727355957, 1.0629069805145264, -0.8751145601272583, -1.4909790754318237, 0.8994554877281189, 1.1057507991790771, -0.12354297190904617, 2.237901210784912, -1.5961710214614868, 1.128529667854309, -0.13647831976413727, 0.07368692755699158, -0.5333710312843323, 1.7962779998779297, 0.5961697101593018, 0.43965959548950195, 0.20928770303726196, 0.5948265790939331, -0.016103804111480713], [-0.240106463432312, -0.764171302318573, 1.3857342004776, -2.0360448360443115, -1.4495102167129517, 0.4217657148838043, -1.8387616872787476, -0.5487409830093384, -0.5920426845550537, -0.1779714822769165, 0.9417865872383118, -0.5286306738853455, -1.884259581565857, 0.10087034106254578, -0.10403729975223541, 0.41323912143707275, 0.2548321485519409, 1.685540795326233, 0.5366340279579163, -1.097488522529602, -0.04444585368037224, -0.7366177439689636, 0.5039963126182556, 1.6308144330978394, -0.013536710292100906, 0.08185127377510071, 1.3518176078796387, 1.1968655586242676, 0.7588168382644653, 1.2987533807754517, -0.7210099101066589, -0.6620099544525146, 1.5023674964904785, 0.06204584985971451, -0.8374970555305481, 0.7719849348068237, 1.9888089895248413, -0.7523757219314575, -2.3931233882904053, -0.575254499912262], [-0.505881130695343, -0.46318814158439636, -0.19086521863937378, 1.13813054561615, 2.0178842544555664, -0.18128544092178345, 0.9180782437324524, 0.01646866463124752, -0.8795397877693176, 1.3820308446884155, -1.4155042171478271, -1.3782436847686768, 0.6334148049354553, 0.2963351309299469, 0.159091055393219, 1.0249677896499634, -1.467509150505066, -1.3537051677703857, 0.3173046410083771, -0.030714306980371475, 0.6097934246063232, 1.7928414344787598, 0.13295234739780426, -0.018080873414874077, 0.32742029428482056, -0.6979357600212097, 1.3282339572906494, 1.8108775615692139, -1.0367488861083984, 0.11417978256940842, 0.08004114776849747, -0.5763692855834961, 0.0026928535662591457, 0.856379508972168, 0.2809590697288513, 1.5253766775131226, 0.5328680872917175, 0.2556225061416626, 1.669342279434204, -0.9785288572311401], [-2.5734126567840576, 0.4144212007522583, -0.5312175154685974, 1.8736451864242554, 0.733805775642395, -1.4907029867172241, -0.9922140836715698, -2.916830062866211, 1.7256802320480347, -1.8124098777770996, 0.42596235871315, -1.7087368965148926, -0.9604936838150024, -1.3528082370758057, 0.9743406772613525, -1.5190789699554443, 0.7958848476409912, 2.120589256286621, 1.0875099897384644, -0.19676539301872253, -0.13600359857082367, -0.17704017460346222, -0.7287533283233643, -0.9925497770309448, 1.2528395652770996, -0.11622226238250732, 0.13197903335094452, -0.7458434700965881, -1.7510615587234497, -2.0056867599487305, -0.927159309387207, -0.86958909034729, -0.44835659861564636, 1.5843040943145752, -2.8967580795288086, 0.3841663599014282, 0.04059284180402756, 2.076842784881592, -2.2950332164764404, -0.9253512024879456], [-0.6344444155693054, -0.49647292494773865, 0.9412304162979126, 0.7224735021591187, 0.1909511834383011, 2.0350258350372314, -0.05743436887860298, 0.5403205156326294, -0.8272526860237122, -0.037748344242572784, 0.1985398679971695, 0.9036284685134888, 1.3361493349075317, 2.1109890937805176, 0.46959561109542847, 1.141894817352295, 0.3629020154476166, -0.12712155282497406, 1.1183717250823975, -0.31130433082580566, 0.41168519854545593, 0.48527470231056213, 1.7357177734375, 0.43515947461128235, 0.31771063804626465, 0.01171904243528843, 0.04262120649218559, 1.570067048072815, 0.7363342642784119, 0.6356618404388428, 1.273963451385498, 0.07381261140108109, -0.40257203578948975, 0.8334531784057617, -1.4362975358963013, -0.7546463012695312, -0.40035098791122437, -0.18017052114009857, 0.2396252453327179, 0.5407978892326355], [1.138203740119934, 0.7055385708808899, -1.3683220148086548, -0.13702300190925598, -0.6859933137893677, 0.9258508086204529, -0.24433965981006622, -0.6395966410636902, -2.320192813873291, 0.8046083450317383, 0.40941327810287476, -1.4059754610061646, -0.5706533789634705, 0.9076555967330933, -1.4023317098617554, 0.8472888469696045, 1.0468688011169434, 0.7276926040649414, -1.179119348526001, 0.7677621841430664, -1.3063818216323853, 1.8595848083496094, -0.6999789476394653, 1.1819884777069092, -0.9789400696754456, -1.010249376296997, -0.04447237402200699, 0.855481743812561, 1.053187370300293, 0.18987531960010529, -0.6784763932228088, 0.8224478960037231, -0.13198642432689667, -1.5497264862060547, -1.5179003477096558, 0.22071462869644165, 1.328566074371338, -0.12513092160224915, 0.9843292832374573, 2.260106086730957], [0.5374724268913269, -0.6847738027572632, -1.1345596313476562, -0.22279734909534454, -0.17038635909557343, 1.1315627098083496, 0.8836796879768372, 0.8276063203811646, 0.15167565643787384, 0.5887473821640015, -0.7374303340911865, -2.176008701324463, -1.5267213582992554, 1.0806076526641846, -0.8039299249649048, -1.857948660850525, -1.6633552312850952, -0.8139783143997192, -1.658719539642334, -0.49894455075263977, 0.20044636726379395, -0.8918384909629822, -0.6907361745834351, -0.37444472312927246, 0.9352678656578064, 0.4941864609718323, -1.2445504665374756, -1.8050198554992676, -0.432391494512558, -0.17886090278625488, -0.41387036442756653, -0.304360032081604, 0.3362489640712738, 0.001437533413991332, 0.048541948199272156, 1.23060941696167, 1.162941575050354, -0.8828973174095154, -0.008098000660538673, 1.69621741771698], [1.0395153760910034, -0.4183720350265503, -0.10593780875205994, 0.8101422786712646, -0.2418304830789566, 1.36676025390625, -0.3561205565929413, -0.7535286545753479, -1.1015676259994507, 0.27628636360168457, 1.3873833417892456, -1.6933280229568481, 0.035585224628448486, 1.2977440357208252, -0.8943073749542236, -1.7913684844970703, -0.7055636048316956, -0.4803388714790344, -0.966706395149231, -0.27443164587020874, 2.042484998703003, -1.3390390872955322, -0.6498852372169495, 0.5082004070281982, 1.0612847805023193, -1.459561824798584, -2.3137876987457275, 0.4054558277130127, -0.22382715344429016, -0.4506097137928009, -1.1580363512039185, 0.7852327227592468, 1.0667684078216553, 1.5130383968353271, 1.6771692037582397, 1.2647392749786377, -0.07946843653917313, 0.025386689230799675, -0.3911287784576416, 0.19300992786884308], [-0.23660080134868622, -0.19574016332626343, -0.001449679839424789, 0.18033255636692047, 0.9133075475692749, -0.6346702575683594, -1.794898271560669, -0.2252086102962494, -0.6512040495872498, 0.4570086896419525, 0.6717479228973389, 0.2973457872867584, 2.2537710666656494, -0.48194053769111633, 0.27244430780410767, 0.6797520518302917, 0.7772363424301147, -0.2789420485496521, -1.005631685256958, -1.8467384576797485, -1.661311388015747, -1.4114162921905518, 1.8070213794708252, 0.45229092240333557, 2.0535659790039062, -0.8173394799232483, -0.5216547846794128, 0.34004342555999756, 1.095723032951355, 0.6703884601593018, -0.6184230446815491, -0.2638642191886902, -1.1208750009536743, -1.8548086881637573, 0.8186302781105042, 0.5452395677566528, -0.4195415675640106, -2.1501660346984863, 1.437627911567688, -1.3248794078826904], [0.7226099967956543, 1.252261757850647, 0.3096734285354614, -1.2842626571655273, 1.717349886894226, -1.0351680517196655, -0.48749956488609314, -1.3344318866729736, -0.8323248624801636, 1.3822706937789917, -1.8073136806488037, -1.2559994459152222, -1.740417718887329, -0.20304226875305176, 0.31664252281188965, 2.50508713722229, 1.1336578130722046, -0.9817456007003784, -0.8481001853942871, -0.7031151056289673, 0.7755588293075562, -1.9640976190567017, 1.6173700094223022, 0.7190392017364502, -0.545150101184845, 0.07513914257287979, 0.10357446223497391, -0.8312984704971313, 0.6506509780883789, -0.5410202145576477, 0.08823990076780319, -0.7321212291717529, -0.8426978588104248, 0.3864423334598541, -0.9078692197799683, -0.4742932915687561, 1.665999412536621, -1.1839925050735474, 0.14713723957538605, 0.444145143032074], [0.8836384415626526, -1.1158043146133423, -0.4698880612850189, -1.2424845695495605, -0.9667893052101135, 0.01821734756231308, -1.0871500968933105, -0.3721911311149597, 0.8524214625358582, 1.440589427947998, -0.15034596621990204, -0.6312990784645081, -0.563575804233551, 0.6192283630371094, -1.174902081489563, 0.13717272877693176, 0.4698760211467743, 0.3215065002441406, 0.073509082198143, -1.6556637287139893, -1.4609800577163696, 0.6843720078468323, 0.5587096214294434, -1.0619539022445679, -1.2898731231689453, 1.179290771484375, 1.2302217483520508, -1.7339239120483398, -1.2319939136505127, -3.181218385696411, -1.4308342933654785, 1.619804859161377, -1.3921953439712524, -1.6991676092147827, 0.13395355641841888, -0.6913002729415894, 1.3745155334472656, -0.5194481015205383, 0.7447259426116943, -0.43438228964805603], [0.21795320510864258, 0.19115811586380005, 0.7701615691184998, 0.3797045946121216, -0.5717440843582153, 0.38491201400756836, 1.182401418685913, 0.07339203357696533, -0.3139501214027405, 0.28077590465545654, 1.2323801517486572, 0.6742447018623352, 0.048228081315755844, 1.4708656072616577, 0.6060011386871338, -0.7991248369216919, -1.2061264514923096, -0.7561098337173462, 0.7566092014312744, -1.8985817432403564, -0.12883874773979187, 0.2765553891658783, -0.7522761225700378, -0.07597467303276062, 1.0241068601608276, 2.108372211456299, 1.2624506950378418, -0.154751718044281, -0.5937036275863647, 0.052547063678503036, 0.7516943216323853, -0.44599610567092896, 0.4361122250556946, 0.367179274559021, -0.6021455526351929, 0.09616418927907944, 0.8526046276092529, -1.895442247390747, 0.20631472766399384, -0.9818319082260132], [0.2096708118915558, -0.23289111256599426, 1.471104621887207, -0.518159806728363, 0.06798936426639557, 0.3881187438964844, -1.371167778968811, 0.06353779882192612, 1.3483896255493164, -0.4395183324813843, 0.4668608605861664, -1.0423086881637573, 1.9764180183410645, -2.1708900928497314, 0.16393139958381653, -1.0982252359390259, 0.6149046421051025, -0.662996232509613, -0.321328729391098, 1.7496883869171143, 0.6579592823982239, 0.6106554269790649, 0.06496874243021011, 0.663239061832428, 1.3515411615371704, 0.18117080628871918, -0.16500897705554962, 0.5215860605239868, 0.28187036514282227, -0.09379246085882187, 1.1017946004867554, 0.6189298629760742, -0.4388706088066101, -0.3750469386577606, -0.5049393177032471, 0.25102725625038147, -0.059272877871990204, -0.6236424446105957, -0.6688414216041565, -0.8783854842185974], [1.3955206871032715, 0.8222006559371948, 0.25551918148994446, 0.710800051689148, 0.06128165125846863, -2.643164873123169, 0.19232411682605743, -1.2684258222579956, 1.2103612422943115, -1.544529676437378, -0.08348562568426132, -0.08726615458726883, 0.2733810245990753, 0.05632796511054039, 0.10840865224599838, -0.13470207154750824, 0.538052499294281, -0.15210554003715515, 0.7092127203941345, 0.7928798794746399, 0.6656717658042908, 1.8011090755462646, 0.6649115681648254, 1.2787550687789917, -1.2523844242095947, -0.6617045998573303, -0.2724342346191406, 1.224973440170288, -2.1070218086242676, -0.7319951057434082, -0.055717598646879196, -0.49314796924591064, 1.1023321151733398, -1.1929939985275269, -0.9107190370559692, 1.2471377849578857, -1.7133458852767944, 0.6778134107589722, 0.5583594441413879, 0.10777514427900314], [-0.7696649432182312, 0.023854104802012444, -1.3737194538116455, 0.10119388997554779, -0.1389184296131134, 0.010164511390030384, -0.2334269881248474, 0.42400845885276794, -0.5538104772567749, 0.10276484489440918, 1.992726445198059, -0.8010584115982056, -0.9704580307006836, 1.5816158056259155, 0.1463685929775238, -0.4192083179950714, 0.40265291929244995, -1.3920538425445557, -2.0054268836975098, -0.6203552484512329, 1.5974537134170532, -0.33724260330200195, -0.3467388153076172, 1.8412015438079834, -0.9039813876152039, -0.3901827335357666, -2.215409994125366, -0.18868021667003632, 0.5984283685684204, -0.1617569476366043, 0.05216019228100777, -0.34605318307876587, 0.7364113926887512, -1.8853956460952759, -1.8814128637313843, -0.9205582141876221, 1.9655230045318604, -0.10440873354673386, 1.0667060613632202, -1.4646204710006714], [0.23027104139328003, 2.6698670387268066, 0.2395310252904892, -1.0970227718353271, 0.7604368329048157, -0.1969890296459198, -1.2434449195861816, -0.434426486492157, -0.7004762291908264, -0.21489283442497253, -0.2755674123764038, -0.6186855435371399, -0.43997088074684143, -2.7824008464813232, 1.069830298423767, 1.657212257385254, -0.6885758638381958, 0.8680946826934814, -1.0473287105560303, -0.6048659682273865, -0.4469413459300995, 0.5870980620384216, -0.5304301381111145, 0.7503702640533447, -1.067510962486267, -0.6759051084518433, -1.0831612348556519, -0.737482488155365, -0.5395777821540833, -0.584333062171936, 1.0874810218811035, -1.2798296213150024, 1.831122636795044, 0.8532336950302124, 0.9147353768348694, -0.9822767972946167, 1.0515124797821045, -0.2668929398059845, 0.3195722997188568, -0.5310194492340088], [1.9116019010543823, 1.1904287338256836, 0.5699159502983093, -1.4052495956420898, -0.07798521965742111, 2.692213296890259, -0.5493464469909668, -1.0919454097747803, 2.3924858570098877, 0.5027385354042053, -0.06624986231327057, -0.8682790994644165, -0.5314428210258484, 0.7979812026023865, 0.6541089415550232, -1.2840163707733154, 0.002180827083066106, 0.6224405169487, -0.8699896335601807, 3.2622110843658447, -0.35367509722709656, -1.1039377450942993, 1.379308819770813, -0.16273906826972961, 0.7872856259346008, 1.1687623262405396, 1.254546046257019, 0.30332183837890625, -0.906745195388794, 0.8583227396011353, -0.7553154230117798, 0.7093045711517334, -0.34365272521972656, -0.8550072908401489, -0.613879919052124, -0.28950241208076477, -0.0023322817869484425, -1.4906681776046753, 0.45544570684432983, -0.30654376745224], [0.31843268871307373, 1.205901861190796, -1.7961972951889038, 2.4175949096679688, 0.061401937156915665, -1.3481162786483765, 0.5887203216552734, 0.6729430556297302, -2.7616682052612305, -0.6796973347663879, 0.6796649694442749, -0.7147355675697327, 0.5868831872940063, 0.22008726000785828, -0.26905661821365356, -0.3464345633983612, 0.03937734663486481, -0.4261056184768677, -0.11811774969100952, -0.8966350555419922, 1.0289502143859863, -0.47568386793136597, 1.7295442819595337, 0.5388000011444092, -0.39090338349342346, 0.505305290222168, -1.2082263231277466, 0.6093758940696716, -0.4821564853191376, 2.3933029174804688, -0.6322903037071228, -0.09054829180240631, -1.2489947080612183, -1.5780671834945679, -1.2959522008895874, -0.4113362431526184, 0.2419097125530243, -0.9054805040359497, -1.904262900352478, -0.39890891313552856], [0.15382055938243866, 0.8682873249053955, 0.2390386015176773, -1.2125027179718018, -0.9425813555717468, 1.1520793437957764, -1.2107785940170288, -0.9102047681808472, 1.1925253868103027, -1.7303906679153442, 1.1459605693817139, 0.6679832935333252, -1.7557971477508545, 0.5877643823623657, -1.275678038597107, 0.7240443229675293, 0.3064923882484436, -0.5013536810874939, 0.37654757499694824, -0.3545938730239868, -0.6739076972007751, -0.6828916072845459, -1.2216358184814453, -0.48847225308418274, -0.009543726220726967, -1.7871291637420654, -0.7043624520301819, -0.8764663934707642, 0.23763908445835114, 0.18915310502052307, -0.9119531512260437, -0.16818715631961823, -1.2265574932098389, -0.9524882435798645, 1.536476969718933, 1.2382783889770508, -1.1086064577102661, -0.29488492012023926, 1.3713432550430298, -1.3658822774887085], [-0.8392662405967712, -0.23283550143241882, 0.07089924812316895, 1.2142481803894043, 1.359683871269226, -0.5377416610717773, 0.7264673113822937, -0.6516844630241394, 0.3951866626739502, 1.2670528888702393, 1.3519412279129028, 1.3014906644821167, -1.1143789291381836, 0.004343677312135696, 1.0042082071304321, 0.14796623587608337, 1.6375285387039185, 0.4468635022640228, 0.5155953764915466, -0.21526260673999786, -0.9963728785514832, -0.23989185690879822, 0.21979643404483795, 1.6683766841888428, -1.138561487197876, -0.2130366563796997, 0.3439050018787384, -0.6017568707466125, 0.9011768698692322, 0.4382789134979248, 1.1814687252044678, 0.644536018371582, -0.4819096028804779, -2.0090556144714355, 0.07578198611736298, -0.016535159200429916, 0.7701719999313354, -0.4142608344554901, 1.9594390392303467, -0.8221356868743896], [-1.2086831331253052, -0.26624467968940735, -0.7567593455314636, 0.7808188796043396, -0.2316453754901886, 0.5230704545974731, 0.30839797854423523, -1.5597931146621704, -1.2894846200942993, 1.2002226114273071, -1.1571128368377686, -0.2778840959072113, 1.452194333076477, -0.4746549427509308, 1.0523200035095215, 1.5241035223007202, -0.9224475026130676, -0.4856131374835968, -0.03550293296575546, 0.4396492838859558, 2.2250046730041504, -1.334788203239441, 0.3519030511379242, 1.4174562692642212, -1.1253671646118164, -0.12737387418746948, -0.5957505702972412, 1.526033878326416, -2.1832034587860107, 0.5980514883995056, 0.33584320545196533, 0.8150705099105835, -0.479667603969574, -0.013967015780508518, 0.675104022026062, 0.12933474779129028, 0.5560373663902283, -0.49917879700660706, -1.5631604194641113, -0.5598838329315186], [-0.04285108298063278, -0.897570013999939, -0.6983470320701599, 0.5502371191978455, 2.055527448654175, 0.3804207146167755, 0.0912155732512474, 1.7531170845031738, 1.058090090751648, -0.591273307800293, -0.47630423307418823, 0.20812711119651794, 1.4248461723327637, -0.5651854872703552, 0.24878931045532227, -1.6496176719665527, 0.03223676234483719, -0.15886887907981873, 0.7731872797012329, -0.732007622718811, 0.8970081806182861, 1.5872160196304321, 0.33996784687042236, 0.8944142460823059, -0.4146151840686798, 0.48470866680145264, 0.45000898838043213, -0.2853294909000397, 0.4484482407569885, -2.0549185276031494, -0.13035301864147186, 1.075879693031311, -0.1913156658411026, -0.2125774472951889, -0.10915728658437729, 0.9953773617744446, 0.34732723236083984, 1.0158928632736206, 0.29128748178482056, 0.008456937968730927], [0.7635794281959534, 0.12416142970323563, 0.8606407642364502, 1.4946386814117432, 1.1577999591827393, -1.490309238433838, 0.657936692237854, 1.0733072757720947, 0.22729341685771942, -0.10863729566335678, 0.6854574084281921, -0.4328778386116028, 0.21070677042007446, 0.9741649031639099, -0.6836348176002502, -0.31936296820640564, -0.07885816693305969, -0.45048996806144714, -0.7088598012924194, -0.46944743394851685, 0.5901381969451904, 0.18280063569545746, -1.0815136432647705, 1.4672369956970215, -0.1473591923713684, 0.6473783254623413, -1.8403528928756714, 0.06755734980106354, 1.190323829650879, -0.2697521150112152, 1.0757824182510376, 0.3611501157283783, 2.288540840148926, 0.9994945526123047, 0.3157872259616852, -0.08202072978019714, -0.47488611936569214, -0.0052071367390453815, -0.1233854666352272, -1.0530967712402344], [0.2777499854564667, -0.3608151376247406, -0.39283064007759094, -2.1187775135040283, -0.1084456741809845, -1.1953014135360718, -0.37074291706085205, -1.1894186735153198, 1.3455365896224976, -1.507729172706604, -1.1331580877304077, 0.8348931074142456, -0.6622031331062317, 0.9836726784706116, 0.002745639765635133, -1.0252289772033691, 0.6266688704490662, 0.9116782546043396, 1.0397658348083496, -0.9631761908531189, -1.5144463777542114, -1.4009697437286377, -1.6884995698928833, -0.6601448059082031, -1.1919574737548828, -0.48630908131599426, -1.1612483263015747, 0.8621684908866882, 0.8090811371803284, -1.4322532415390015, -0.4709632992744446, -1.1859192848205566, 1.8253883123397827, -0.07350145280361176, -0.22011792659759521, 0.3325020670890808, 0.5084868669509888, 0.6164084076881409, -0.06271907687187195, 1.2404075860977173], [-0.12604139745235443, -0.39526939392089844, 1.7607147693634033, -0.014631757512688637, 0.3153347671031952, -1.7309226989746094, 0.8912745118141174, -0.41408804059028625, 0.6579246520996094, -0.7926488518714905, 0.8465644121170044, -1.1723910570144653, 0.5128763914108276, 0.3870184123516083, -1.1639609336853027, -0.26475030183792114, 0.9215399622917175, -0.2397080808877945, 0.1583271026611328, -0.36602622270584106, -0.4335739314556122, 1.0269755125045776, 0.014635954052209854, 1.0602025985717773, -0.5137643814086914, 0.1376538872718811, -1.0042680501937866, 0.8579918742179871, 2.5581002235412598, 0.34627386927604675, 0.02445370890200138, 0.8705883622169495, -1.2716636657714844, 0.06617899239063263, 1.5905685424804688, -1.0160835981369019, -0.43383029103279114, -0.46085819602012634, -0.9549076557159424, -0.9261330962181091], [1.1727839708328247, -0.038352854549884796, 1.1041333675384521, -0.35146763920783997, -1.3584812879562378, -0.022971676662564278, -1.063720703125, -0.4912058115005493, 0.10923442244529724, 0.13568401336669922, 1.1581814289093018, 0.4538831412792206, -0.04681660979986191, -1.2575922012329102, -0.5389209985733032, 0.9089289307594299, 0.7833478450775146, -1.0533422231674194, -0.11670457571744919, -0.0031509893015027046, -0.028465529903769493, 0.3051166236400604, 0.20028968155384064, -0.8274825811386108, 0.10386240482330322, 0.1725335270166397, 1.2860908508300781, -0.6170153021812439, 0.07749754935503006, 1.0894196033477783, 1.1633172035217285, 0.5041707158088684, -2.932957172393799, 0.19298118352890015, 0.24787531793117523, 0.15724970400333405, 1.708286166191101, 1.2172434329986572, 0.4996461868286133, -0.5654078125953674], [-0.4195399582386017, -1.9668877124786377, -0.21863523125648499, -0.745758056640625, 0.7275048494338989, -0.508994996547699, 0.05833504721522331, -2.481189727783203, 0.4366045296192169, 1.0683846473693848, 1.2472708225250244, -0.5360559821128845, 0.0006394741358235478, -1.130868911743164, -0.7315441966056824, -1.6156333684921265, -0.2053438425064087, -0.2356247752904892, 0.2769469916820526, 0.1153516098856926, -1.5647058486938477, -0.07111331075429916, -0.29935136437416077, 0.6971368789672852, 0.8489539623260498, -0.18643558025360107, 1.297751545906067, 0.3186829388141632, -1.1599067449569702, -0.9190930128097534, -0.5876703858375549, -1.008449673652649, 2.463008165359497, -0.054370392113924026, -1.3150638341903687, -0.9281361103057861, 1.3829245567321777, 0.8723655939102173, 1.4153611660003662, 0.36042532324790955], [-0.20433585345745087, -0.6424480080604553, -1.2454127073287964, -0.25257664918899536, -0.2940731346607208, 0.823699414730072, 1.9394546747207642, -1.0640010833740234, -1.874512791633606, 0.06277710944414139, 0.22646072506904602, 0.5393387675285339, 0.492129921913147, -0.36402690410614014, -1.6918272972106934, -0.13496042788028717, 0.22847425937652588, -0.25390464067459106, -0.7211759090423584, -0.7798030972480774, 1.4220370054244995, -1.145216941833496, 0.06142221391201019, 1.0515927076339722, -1.1342073678970337, -0.38480144739151, 0.33538562059402466, 0.44028374552726746, -1.0077996253967285, 0.6398400664329529, -0.008369521237909794, -1.2678511142730713, 1.2957885265350342, 0.8870998620986938, 1.0585883855819702, -0.5471820831298828, 0.6059858202934265, 0.1826719492673874, 0.26606547832489014, 3.2377610206604004], [0.30946311354637146, 1.227561116218567, 1.093997836112976, 1.7014669179916382, -1.4305301904678345, -0.28668704628944397, 0.0006429495406337082, -0.2539307475090027, -2.057497262954712, -0.21777556836605072, -0.17322596907615662, 1.005338191986084, -0.43042755126953125, -0.3953694701194763, -0.4915419816970825, -1.0592187643051147, -1.2449454069137573, -0.7119722366333008, -0.841917872428894, -0.11920200288295746, -0.35155189037323, 0.7869924902915955, -0.9310600757598877, -0.4903471767902374, -1.7929667234420776, -0.07962621003389359, -1.1362550258636475, 0.09943357110023499, 0.1441628336906433, 0.07509981095790863, -0.6652200222015381, -1.2525938749313354, 0.23991942405700684, 0.48477989435195923, 0.3178479075431824, 0.13970717787742615, -0.5530731081962585, -0.5193670988082886, -0.90662682056427, 0.30350714921951294], [0.4674544632434845, -0.39475223422050476, 0.8234648108482361, -0.6513001918792725, 0.08775141090154648, -2.247065305709839, -0.35902440547943115, -0.836762547492981, 1.4742491245269775, -0.15554934740066528, 1.8642961978912354, 1.4937249422073364, 0.4082409739494324, -0.8248592019081116, 0.9199091792106628, -0.9553229212760925, -0.16689574718475342, 0.12181361764669418, 0.4484948515892029, -0.17799164354801178, 0.9161752462387085, -0.708074688911438, 0.7318195104598999, 1.624048113822937, -1.457299828529358, -2.0229430198669434, -1.1681853532791138, -0.7997605204582214, 1.278518557548523, -0.036196861416101456, 0.6518714427947998, 0.6442152261734009, -0.4503195285797119, -1.6235476732254028, 1.0926382541656494, 0.4071897566318512, 0.9399310946464539, 0.28810742497444153, 0.6762294769287109, 1.2928094863891602], [0.06757897883653641, 0.7531164884567261, 2.130568027496338, 0.525601327419281, 0.22933709621429443, -0.5291791558265686, 0.0025630644522607327, 1.0647375583648682, -1.4928340911865234, -1.1508684158325195, -1.2127201557159424, 0.15974710881710052, -2.2406575679779053, 0.39954838156700134, 0.7302969694137573, -0.5938206911087036, 1.2217546701431274, 0.9056586027145386, -0.3840238153934479, -1.421494722366333, 0.649513304233551, -0.38791796565055847, -1.7742184400558472, 1.4864860773086548, 0.8056811690330505, -0.8496320247650146, 1.66191828250885, 0.8234502077102661, -0.2894810736179352, 0.21733251214027405, 0.27973660826683044, 1.6987881660461426, 0.60893714427948, 0.8955973386764526, 1.4841209650039673, 0.820606529712677, 0.10543286055326462, -0.8335912823677063, -2.4267706871032715, 1.6684900522232056], [0.2266232669353485, -0.939264178276062, -0.8697306513786316, 1.1892821788787842, 0.6260943412780762, -0.42997825145721436, -0.9064950942993164, 1.0890010595321655, 0.8578160405158997, -0.7170575857162476, -0.5768099427223206, -2.628410816192627, 0.43369951844215393, -1.1767487525939941, 0.849048912525177, 1.3808531761169434, 0.4763244390487671, 0.9811273813247681, -0.8928436636924744, -0.5811675190925598, 0.4261598587036133, 1.0578893423080444, -0.09022700786590576, 0.13695241510868073, 1.1559478044509888, 0.47089624404907227, 0.5315815806388855, -1.8257157802581787, 1.0852640867233276, -1.3431748151779175, -1.5082106590270996, 0.7932571172714233, -0.11068817228078842, 0.4813157320022583, 1.0244355201721191, -0.043615274131298065, -0.07938431948423386, 1.3324638605117798, -3.1622345447540283, -0.04113111272454262], [0.2635657489299774, -1.4998880624771118, -1.709851622581482, 0.002062359591946006, -0.0478006936609745, -1.8417041301727295, 0.12150714546442032, 0.15272113680839539, 0.5877599716186523, -0.8648727536201477, -0.4664088785648346, 0.45174160599708557, -0.4830182194709778, -1.319350242614746, 0.9184253811836243, -0.3100596070289612, -0.2474081665277481, 1.5128276348114014, -2.231363534927368, 2.283071517944336, -0.6900027990341187, -0.35741278529167175, 1.8814804553985596, -1.0967214107513428, 0.9299072027206421, -0.062230173498392105, -0.45506250858306885, 1.1720513105392456, -0.6500871777534485, -1.3379707336425781, -0.9822876453399658, 0.49001744389533997, -1.3188577890396118, -0.4553517699241638, -0.25422847270965576, -0.5510448813438416, -1.1210112571716309, -0.4980420172214508, -0.19533805549144745, 0.7359302043914795], [0.385050505399704, -1.3271152973175049, -0.17587853968143463, 1.6475321054458618, -1.7858136892318726, 0.988993227481842, -1.08025324344635, -0.7282124757766724, 0.6691267490386963, 1.4090642929077148, -0.6956024169921875, -0.6878671050071716, 0.5698267817497253, -1.0890997648239136, 0.14757123589515686, 1.0604221820831299, -0.05529779568314552, 1.6944559812545776, 0.15173260867595673, 0.24455516040325165, 0.14233598113059998, -0.18702080845832825, -1.1480532884597778, 0.9426551461219788, -0.23777304589748383, 1.7118338346481323, 2.375964641571045, -0.2314685583114624, 1.9662160873413086, -0.039025742560625076, -0.5173557996749878, -0.5220304131507874, 0.2632013261318207, -0.18311823904514313, 0.9162387251853943, 0.5912405848503113, 0.36243727803230286, 1.115015983581543, 2.744441270828247, -1.2519835233688354], [-0.3500833809375763, 0.21378149092197418, -0.5325168967247009, 0.2805231511592865, -0.5497006177902222, -1.0534021854400635, 0.5940044522285461, -0.05052003636956215, 0.030010327696800232, -0.7954552173614502, 0.366226464509964, 0.4505176544189453, -0.370134562253952, 1.060782551765442, 0.5499166250228882, -0.36875855922698975, -1.1100637912750244, -0.45383262634277344, -0.1698407381772995, 1.620369791984558, 0.3785818815231323, -0.6334461569786072, -2.0031397342681885, -0.9662157893180847, 0.6032651662826538, -0.29141974449157715, 1.61708402633667, -0.15538664162158966, -1.672358512878418, -1.6986008882522583, -0.4904465973377228, 1.8321101665496826, 0.8613865971565247, 0.9979071021080017, 0.839760959148407, 0.9124314785003662, -0.12835679948329926, -0.6488124132156372, -0.42705297470092773, 1.9465135335922241], [-0.41903990507125854, 1.5178554058074951, -0.011956200934946537, 0.6156187653541565, 0.4619872570037842, 0.18992948532104492, 1.136630654335022, 0.48633521795272827, -0.4071362614631653, 1.3793420791625977, -0.6314230561256409, -0.7337749004364014, -2.880873203277588, 1.5346397161483765, -1.5516510009765625, -1.6800953149795532, 0.21247710287570953, -1.077459692955017, 0.6847582459449768, 0.3552769720554352, 0.055673930794000626, -0.5267870426177979, -1.0748564004898071, -0.24180114269256592, -0.7948473691940308, 1.3182435035705566, 0.14158841967582703, -1.8329921960830688, -0.251552551984787, -0.7736738324165344, 2.567146062850952, -1.56987464427948, 0.07788659632205963, 0.09410516172647476, 1.7617965936660767, -2.6847658157348633, -0.1869610995054245, 0.6538917422294617, -1.041547417640686, 3.2060487270355225], [1.4316496849060059, -0.5461799502372742, -0.8774176836013794, -0.04976922273635864, 0.13415764272212982, 0.6035465598106384, -0.3057504892349243, -0.17775538563728333, 0.2448401153087616, 0.4780324101448059, 1.243562936782837, 0.8528338670730591, -0.37361758947372437, 0.06501162797212601, 0.5781627893447876, -0.7047868967056274, 0.5067717432975769, 0.43034082651138306, 1.503643274307251, -0.6681105494499207, -0.7898460030555725, -0.5340665578842163, 0.08667019009590149, -1.2473851442337036, -0.15698383748531342, -1.3539620637893677, -1.001463532447815, 0.8647684454917908, 0.06062823906540871, 0.3998086452484131, -0.14073044061660767, 1.498225212097168, -0.44898444414138794, 1.088842749595642, 0.5862566232681274, -1.5843102931976318, -1.514648675918579, 0.10781700164079666, 0.48556289076805115, 1.279563069343567], [-2.60841965675354, 0.09880205243825912, 0.06147424504160881, -0.8372069001197815, 1.0587483644485474, 0.9754952192306519, -0.9241894483566284, -0.28207311034202576, 0.1607646644115448, -1.1765772104263306, -1.0251775979995728, 0.11475127935409546, 0.1325143575668335, -0.6787518262863159, -0.4049929082393646, 1.408557653427124, 0.9163420796394348, -0.12013164907693863, 0.5669687390327454, 0.5495359897613525, -0.6519277691841125, 0.3027576506137848, 0.37728071212768555, 0.35994720458984375, -1.0904316902160645, -0.9500486254692078, 0.37104418873786926, -1.7543350458145142, -0.4072745740413666, -0.42494744062423706, 2.0701165199279785, 1.4970221519470215, 0.44825688004493713, 0.504757285118103, 0.9220305681228638, 0.43829336762428284, -0.32950037717819214, -0.27785825729370117, -1.19899320602417, 0.15380188822746277], [0.7102493643760681, 2.0777370929718018, 0.3636152744293213, 0.49144428968429565, 1.2007197141647339, -0.4080043137073517, 0.12823761999607086, 1.032546877861023, -1.5603702068328857, -1.0094428062438965, 1.8127425909042358, -1.5994887351989746, -0.05962646007537842, -0.05450667068362236, -1.453126072883606, 1.0774004459381104, -0.8478930592536926, 1.826791524887085, 0.06824661046266556, 3.460616111755371, 1.2498905658721924, -0.2242165356874466, -1.6665719747543335, -1.534397840499878, 0.19884541630744934, -0.45120495557785034, -1.0907843112945557, 0.002547239186242223, -0.37562981247901917, 0.8390304446220398, 0.744024932384491, 0.22143957018852234, 1.5741991996765137, 0.4065168499946594, -0.2587457299232483, -0.8733287453651428, -0.9156059622764587, 0.03861866518855095, 1.5275906324386597, -1.3025028705596924], [-0.1721724420785904, -0.8438998460769653, 0.1589987426996231, -0.1629641056060791, -0.15696202218532562, -0.6345286965370178, -1.047263503074646, -0.4139491021633148, 0.2732544541358948, -0.40317052602767944, 0.9889949560165405, 0.34044620394706726, -0.33894428610801697, 1.5591456890106201, -1.2883356809616089, -0.24422955513000488, 0.48503348231315613, -0.12471786141395569, 0.9969561696052551, 0.0020174176897853613, 1.4232072830200195, -0.8812808394432068, 1.2400493621826172, 0.050160687416791916, -1.1791977882385254, -1.2405064105987549, 0.23017054796218872, 1.1063897609710693, 1.2482030391693115, -1.6049057245254517, 1.1559863090515137, -0.10431359708309174, -0.6283493638038635, 0.2314741611480713, -1.3182003498077393, 0.4681869149208069, 1.2289537191390991, 1.1262396574020386, 0.9245116114616394, -0.03698974847793579], [-0.12621822953224182, -0.9440898299217224, -1.0404396057128906, 0.4777868092060089, 0.3759846091270447, -0.46935099363327026, 1.7403520345687866, 1.8325390815734863, 0.29925355315208435, 0.24554723501205444, 0.4401656985282898, -0.7645598649978638, 0.9897030591964722, -0.39570432901382446, -0.6473767757415771, 0.3939042091369629, 0.5235522985458374, -0.0636824294924736, 0.4606175720691681, 0.3459533452987671, 0.14406199753284454, 0.1063719168305397, 0.984247624874115, 0.5828583240509033, -2.268996238708496, -1.341647982597351, -0.035011470317840576, -1.0624080896377563, 1.071804404258728, -1.4617581367492676, 1.2072622776031494, -0.9073344469070435, -0.006002530921250582, -0.3011671006679535, 0.7076659798622131, -1.053814172744751, 0.7416713833808899, -0.9609587788581848, -0.11872151494026184, 0.5352702140808105], [0.28820863366127014, 0.2651212513446808, 0.027320386841893196, 1.8642001152038574, -1.2139350175857544, -3.2328600883483887, 0.4431057572364807, -2.9159858226776123, -0.6729657053947449, 2.188873052597046, 1.0818920135498047, 0.8279663324356079, 0.9006506204605103, 1.6508899927139282, -1.576797366142273, -0.2243305891752243, -1.0342987775802612, -0.9556640982627869, 0.23749880492687225, -0.7802298665046692, -0.1112305074930191, 0.6438028812408447, -0.8193600177764893, 1.7477245330810547, -0.42976611852645874, 0.7530232071876526, -0.8313443064689636, 1.7188969850540161, 0.6150806546211243, 0.045583829283714294, -0.9394689202308655, -1.7822695970535278, -1.2978382110595703, 1.2222954034805298, 0.6350851655006409, 0.1651798039674759, 0.13523522019386292, 0.07970725744962692, 1.4094140529632568, -0.42495518922805786], [-0.008553626947104931, 0.04120248183608055, 0.07817400246858597, 0.1904914677143097, -0.16654108464717865, 0.5103368759155273, -1.0005828142166138, 0.11550211161375046, -0.5635918378829956, 0.7090721130371094, 0.9175035357475281, -0.12116318196058273, -0.8278622031211853, -0.21353667974472046, 0.4661217927932739, 0.10274495929479599, -0.9189772009849548, 0.9855949878692627, 1.4349122047424316, -0.662621021270752, -0.3519585132598877, 0.16500574350357056, 0.714177131652832, -0.372196763753891, -0.8781946897506714, 1.0847828388214111, 0.1297202855348587, 1.2330704927444458, -0.7348998188972473, -0.6228815913200378, 0.04156181216239929, 1.639625906944275, 0.3162272870540619, -1.094126582145691, -0.6680351495742798, -0.2971762716770172, -1.5479289293289185, 1.0151668787002563, 0.7657147645950317, -0.488294392824173], [-0.5797070264816284, -0.8704679608345032, -0.9977656006813049, 0.5594545602798462, 0.7951507568359375, 0.8957397937774658, -0.34856128692626953, -0.1333492547273636, -0.43622642755508423, -1.2868139743804932, -0.005596878007054329, 0.8945061564445496, 0.02664772793650627, 0.4394299387931824, -0.12345359474420547, 0.18788397312164307, 0.03632232919335365, 0.7348951101303101, -1.100292682647705, -0.6053658127784729, 0.7925248146057129, -0.037984855473041534, 0.48710259795188904, -0.5542372465133667, 0.3902055323123932, -0.9415680766105652, 0.6343588829040527, 0.8036790490150452, 0.3522700071334839, 0.6599216461181641, -1.0003669261932373, 1.8849351406097412, -0.44864368438720703, -0.2858857214450836, -0.476816326379776, 0.847678542137146, -0.6385942101478577, 1.2222669124603271, 1.017547845840454, -0.01603473536670208], [-0.032223787158727646, 0.36018168926239014, 0.4519903063774109, 0.6163963079452515, 0.9775081872940063, -0.4236781597137451, -1.6591246128082275, 0.16456809639930725, 0.996410071849823, 1.2481919527053833, 0.6189218163490295, -0.4011165499687195, -0.8687145113945007, 0.37832146883010864, -0.4805510342121124, -0.8710014224052429, -0.5740487575531006, 0.749922513961792, 0.9653598666191101, -0.7888842225074768, 2.37752366065979, -1.220198392868042, -0.5511966347694397, 0.7577733993530273, 0.7912462949752808, 0.3599017262458801, -0.5940961241722107, -0.30306074023246765, -0.8495022654533386, 0.23125746846199036, 0.3407534062862396, -1.4792286157608032, -0.5736820697784424, 1.3400959968566895, -1.1411406993865967, -0.4807440936565399, -0.06591715663671494, 0.12576206028461456, -0.8691807389259338, 0.1271248310804367], [1.5211008787155151, -0.2950303852558136, -0.9820193648338318, -0.02424219809472561, 1.3238637447357178, 0.6137252449989319, 0.16065825521945953, -1.1551436185836792, -1.6511976718902588, -0.4865739941596985, 0.8591392040252686, 0.1427379846572876, -0.5134857296943665, -1.1587762832641602, 0.30862918496131897, -0.6613583564758301, -0.04149191454052925, -1.009301781654358, 1.1798779964447021, 0.8291152715682983, 1.9164221286773682, 1.1151889562606812, 1.454602599143982, -0.00015915297262836248, -1.4863237142562866, 0.005790308117866516, 0.0639941394329071, 1.1387912034988403, -0.05976760387420654, -0.7935509085655212, -1.116028904914856, 0.09432132542133331, 0.5417704582214355, 0.5968458652496338, 0.09259022027254105, -1.7309870719909668, -1.1776267290115356, 1.8011491298675537, -1.3054742813110352, 0.49464404582977295], [0.12012535333633423, -0.4645244777202606, 0.9676666855812073, -0.28049004077911377, 0.22161982953548431, -0.9637848734855652, -0.7947887778282166, 2.1938071250915527, 0.2720426321029663, -0.32883936166763306, 0.32284119725227356, 1.968077301979065, -0.6738249063491821, 0.6457188129425049, 0.5207985043525696, -1.2095946073532104, -0.40392252802848816, 0.1450626403093338, 0.5815805196762085, -1.603162169456482, -2.187166929244995, 0.1045236587524414, -0.3398978114128113, 1.8050082921981812, 1.0501517057418823, 1.426694393157959, 1.3460954427719116, -0.6534709930419922, 1.8003875017166138, 0.3792110085487366, 0.43864166736602783, 0.17435091733932495, 0.15994694828987122, -0.5145165920257568, -0.2678242027759552, -0.3177584707736969, 1.0406579971313477, 0.09911302477121353, 1.799726963043213, -1.564491868019104], [-0.13117501139640808, 2.8095901012420654, -0.20655155181884766, -3.160806655883789, 1.2325903177261353, -0.28702229261398315, 0.02046528458595276, 0.319124698638916, -0.7560021281242371, 0.11221840232610703, 0.5097734928131104, -0.019406383857131004, 0.12696918845176697, -0.3675532042980194, -0.11329039931297302, 0.0991070568561554, -0.19356617331504822, 0.19695483148097992, 1.6158194541931152, 1.5338757038116455, 1.693095326423645, -0.7179567813873291, 0.12692810595035553, 0.8592362999916077, -0.03588525950908661, -0.6847205758094788, 1.5258004665374756, -1.6108890771865845, -1.652315616607666, -0.49977603554725647, 0.48006680607795715, -0.5520890951156616, -0.5497025847434998, 0.6220149993896484, 0.08302103728055954, -1.0325621366500854, -0.4524648189544678, 0.5179322361946106, 1.0121221542358398, -0.8199805021286011], [-1.3036302328109741, -0.998974621295929, -0.4877217710018158, 0.6133495569229126, 0.02054060436785221, -0.6614226698875427, 0.09867410361766815, 0.23321160674095154, -0.030717406421899796, -1.4173365831375122, -0.25670599937438965, 1.3811583518981934, -0.5824722051620483, -0.851401686668396, -0.3028412461280823, 0.45013681054115295, 1.4682339429855347, -0.45435211062431335, 1.1579324007034302, -0.5644537210464478, 0.605797290802002, -0.34055063128471375, -1.0978084802627563, 0.05917222797870636, 1.570501446723938, -0.5325204133987427, 0.4703361392021179, -0.20163127779960632, 0.14507412910461426, -0.2189687192440033, 0.02434510923922062, -1.3806707859039307, 0.27096959948539734, 1.560238003730774, 0.6055808067321777, 1.226454257965088, 1.5555657148361206, 1.2043417692184448, -0.37676355242729187, 0.09294351935386658], [-1.1539143323898315, 2.1744747161865234, -0.7479304075241089, 0.5082933902740479, 0.797353208065033, -2.1029345989227295, -0.15280529856681824, 1.9063537120819092, 1.6031073331832886, -0.009271799586713314, 0.7802503108978271, 0.2528882622718811, 0.40857386589050293, -0.06852617859840393, 0.007276453077793121, -0.7155063152313232, -0.8204634785652161, 0.7803009152412415, 0.9546681046485901, -1.7773278951644897, -2.223040819168091, -0.5999859571456909, -0.5178037881851196, -0.05860200896859169, -0.3728553354740143, 1.150266408920288, -0.31487008929252625, -1.0147435665130615, 0.01873626932501793, 0.9946292042732239, -0.6566722989082336, -0.3803088068962097, -0.9363106489181519, -0.40555906295776367, 0.281559020280838, 1.4520235061645508, 1.177827000617981, 0.8366790413856506, -0.9828917980194092, 0.6274341344833374], [0.5534022450447083, 1.1840991973876953, 1.4829429388046265, -0.5158526301383972, 0.11730103194713593, 0.16156545281410217, -0.7831987738609314, -0.679358184337616, 0.7470472455024719, -0.03575935959815979, 1.3059884309768677, -0.1606157273054123, 0.041149355471134186, -0.7503010034561157, 1.8597439527511597, 1.3029024600982666, 2.4992456436157227, 0.23610007762908936, -0.38951095938682556, 1.8976678848266602, -1.8334873914718628, 0.40858185291290283, -0.0922975242137909, -0.29602131247520447, 1.116737961769104, 0.17257723212242126, 0.3638622462749481, 0.5883321166038513, 0.8100450038909912, -1.1259828805923462, 0.9778321385383606, -0.7274112701416016, 1.6451044082641602, -0.6377733945846558, -1.0102243423461914, -0.27556878328323364, 0.042684778571128845, 0.041075076907873154, 1.222281813621521, -0.3418482542037964], [0.42453864216804504, -0.632400393486023, -0.12608249485492706, 0.979115903377533, 0.29207709431648254, -0.017583481967449188, -0.28928136825561523, 0.8001984357833862, -1.4767236709594727, 1.9374662637710571, -0.13524633646011353, -1.650660753250122, -1.505048394203186, 0.7891109585762024, 1.3423858880996704, -0.5355232357978821, -0.29145413637161255, -0.4457085132598877, 0.770771861076355, 0.011138498783111572, 0.5186856985092163, 0.7550776600837708, -0.4274158477783203, 1.6913210153579712, 0.6922219395637512, -0.41664254665374756, 1.0480607748031616, 0.5881930589675903, -0.5508432388305664, -0.9746454358100891, -0.768731415271759, 0.8497254848480225, 0.5249671936035156, -2.267221212387085, 0.028173742815852165, 0.12750303745269775, 0.49822527170181274, 0.5405477285385132, 0.8559388518333435, 0.426456481218338], [-0.6360512375831604, 0.1898987591266632, -0.4895995557308197, 1.4694247245788574, -0.5985209345817566, -0.03734968230128288, 0.9449252486228943, -0.19635406136512756, 0.9658570289611816, -1.45842444896698, 0.7102567553520203, 1.2041701078414917, 0.6744245290756226, 1.7563894987106323, -1.0929023027420044, -1.8590329885482788, -1.0918223857879639, -0.7409998178482056, 0.6184588074684143, -0.8460698127746582, 0.21102984249591827, 1.2837008237838745, 0.75263911485672, -0.9359577894210815, -0.06601426005363464, 0.40630605816841125, 0.905828595161438, -1.0872626304626465, -1.6425172090530396, 0.8698185086250305, 0.09779006987810135, 1.5353026390075684, -0.5828877091407776, -0.9924148321151733, -0.8898462057113647, 0.3038904070854187, 0.31228411197662354, -1.3120899200439453, -0.00448809377849102, 1.4277594089508057], [-0.6854726076126099, 2.2890031337738037, -0.4515264928340912, -0.21334196627140045, 0.0490964911878109, 0.9807898998260498, -0.9871026277542114, 0.6122892498970032, 1.955228328704834, 1.5528393983840942, -0.21539396047592163, -0.6723935604095459, 0.7469809651374817, -0.8868414163589478, 0.552422046661377, 2.0414164066314697, -0.6634613275527954, -0.0862625315785408, 0.602565348148346, 0.8505273461341858, 0.9735807180404663, -0.32294371724128723, -0.7911534309387207, 1.1551162004470825, -0.08041661977767944, -0.6442010998725891, -0.37271106243133545, 0.370924711227417, -0.5723676085472107, -0.8535189032554626, -0.45604050159454346, 0.7764198184013367, -2.007655620574951, -1.006258487701416, -1.5970730781555176, 0.9401450753211975, 0.8547153472900391, -2.0477170944213867, -0.9797509908676147, 0.5686781406402588], [-0.40685856342315674, 0.962266206741333, 1.3389829397201538, -0.47783300280570984, 0.02317926101386547, 1.8033716678619385, 0.26256677508354187, 0.6106985807418823, -1.4303321838378906, 1.3987782001495361, -0.6572785973548889, 0.10738953948020935, -0.7290887832641602, 0.7801632285118103, 0.712780773639679, -0.5050926804542542, -1.260654330253601, -1.0304549932479858, -1.2819396257400513, -0.7863902449607849, 0.21037517488002777, 0.618855357170105, 0.31647732853889465, 0.4235483705997467, 0.5293722152709961, -0.4171506464481354, 0.8651429414749146, -0.04107796028256416, 1.3959064483642578, -1.2017827033996582, 1.1134705543518066, -0.8468949198722839, 2.144557476043701, -0.08799570798873901, 0.04140661656856537, 0.7000932693481445, 1.0502272844314575, -0.30298012495040894, 0.3047289252281189, -0.9662452340126038], [0.40839409828186035, -0.6572247743606567, -0.7753473520278931, 0.995444655418396, -0.19120632112026215, 1.5147467851638794, -0.8357108235359192, 0.7216821312904358, -0.855744481086731, 1.7657023668289185, 1.4814121723175049, 1.216655969619751, -0.9185689091682434, -0.7440938353538513, -0.519866406917572, -0.6431766748428345, -0.10646232962608337, -1.3882609605789185, -0.5729863047599792, 0.42145660519599915, -1.0812658071517944, -0.5459819436073303, 0.3681713342666626, 0.15737417340278625, -0.4559224843978882, -0.8427727222442627, -1.1299159526824951, -0.62702476978302, -0.3313765525817871, 1.6144392490386963, 0.20025691390037537, -0.5801483392715454, -2.088524103164673, -1.623218297958374, -1.3940296173095703, -1.6038748025894165, -0.31272780895233154, -1.0584008693695068, -0.37697166204452515, 1.3628002405166626], [-0.29395565390586853, 0.3252798914909363, 2.419875144958496, -1.4351754188537598, -2.0318398475646973, -0.21882429718971252, -0.7143633365631104, -0.7839030027389526, -0.150905579328537, -0.375846803188324, -0.4043722450733185, 1.1393355131149292, 2.497605800628662, -0.4465000033378601, 0.9316411018371582, -0.8986678719520569, 0.8598236441612244, 1.2692346572875977, 0.026670075953006744, 2.319946527481079, 0.7829316258430481, 1.618499517440796, -1.4305126667022705, 0.39799726009368896, 1.8532226085662842, -0.9574381113052368, -0.8447225093841553, -0.24179117381572723, -0.08444083482027054, -1.2873826026916504, -0.30198800563812256, -2.133445978164673, -0.3186750113964081, 0.045650649815797806, -1.3368000984191895, 0.6960963606834412, 0.3812769055366516, 0.42667263746261597, -0.14840808510780334, -0.40870025753974915], [0.10860192030668259, -0.37873777747154236, 1.1767314672470093, -0.6941757202148438, 0.09361390769481659, -2.0840163230895996, 0.21222642064094543, 1.4489307403564453, 0.5185114145278931, 0.8884501457214355, -1.2564754486083984, 0.07773976773023605, 0.20808261632919312, 1.8451242446899414, 0.9860721826553345, 2.166551113128662, -1.776951551437378, -1.8045684099197388, -1.0073878765106201, 2.680854558944702, -0.8716496825218201, -0.13752774894237518, 0.9509302377700806, 0.07733319699764252, 0.8148366808891296, 0.019419189542531967, -0.7156179547309875, -2.312067985534668, 0.21717506647109985, -0.28731560707092285, 1.6113834381103516, -0.9478505253791809, 1.4050657749176025, 0.8166813850402832, -0.6595598459243774, 0.44696757197380066, -1.4020321369171143, -0.5741624236106873, 1.4217318296432495, -0.4958553612232208], [-0.0700434148311615, -1.2168145179748535, 1.4886329174041748, 0.5045151710510254, 0.28800123929977417, 0.18167634308338165, 0.030820004642009735, 1.5392075777053833, -1.0059590339660645, 0.9260400533676147, -0.380392462015152, -1.406577229499817, -0.43077537417411804, 0.2510381042957306, -0.5226773619651794, -0.41686561703681946, -1.1759401559829712, 0.4350012242794037, 0.5703017115592957, 0.7172683477401733, -0.9322837591171265, 0.05100297927856445, 0.2931273579597473, -1.233175277709961, 1.8397828340530396, 0.8570986986160278, 0.8034219741821289, 1.1635093688964844, 0.37809956073760986, 0.14352719485759735, -1.3104387521743774, 1.0955684185028076, 0.5554284453392029, -0.7480483651161194, -0.8033018708229065, 1.1665124893188477, 0.9522474408149719, -0.37739866971969604, -1.0449295043945312, -0.9093427062034607], [1.614948034286499, 0.7753204703330994, -0.34003961086273193, -1.9852569103240967, -0.1609882265329361, -1.5608460903167725, 1.951390027999878, -0.33476030826568604, -0.4115614891052246, 0.9776837825775146, 0.8782769441604614, -0.866354763507843, -0.11510856449604034, 0.564736545085907, 0.03940640017390251, -1.6647294759750366, -1.0167081356048584, 0.016554825007915497, -0.8920838236808777, 1.6731737852096558, 1.8483346700668335, 1.2806345224380493, -0.9098228812217712, 0.3851796090602875, 0.44962823390960693, -0.21854883432388306, -0.841802716255188, -0.030192425474524498, 1.5360844135284424, -1.5154905319213867, -1.1745569705963135, 2.5198264122009277, 1.1020643711090088, 0.17481592297554016, -1.5569261312484741, -0.7965101003646851, -0.9943341612815857, -0.5635052919387817, -1.0674716234207153, 1.0210182666778564], [1.5797654390335083, 0.16284292936325073, 1.509459376335144, -1.0252467393875122, 0.1582162231206894, 0.640958845615387, 0.7954806685447693, 0.1804884970188141, 0.13067986071109772, -0.4903784692287445, -0.19983099400997162, -0.902446448802948, 1.1234842538833618, 0.18126438558101654, 0.1170826107263565, 0.6017191410064697, -1.018458366394043, 2.412428617477417, -0.33858367800712585, 0.018086755648255348, -1.8502863645553589, -0.775740921497345, -0.22721116244792938, -1.5268546342849731, 0.6291545629501343, 0.614939272403717, 0.8109269142150879, -0.36347517371177673, 0.16778334975242615, -1.4783228635787964, 0.5616063475608826, -1.0317155122756958, -0.87096107006073, -0.6420374512672424, 1.1100701093673706, 1.3308054208755493, 0.4331311881542206, -0.38427069783210754, 1.0594761371612549, 0.8666236400604248], [-1.7771074771881104, -0.9231175780296326, -0.5076017379760742, 0.614852786064148, 0.42121458053588867, -0.28354883193969727, 0.5105491876602173, 1.6610907316207886, -0.25520968437194824, 0.8656473159790039, -0.059070736169815063, 0.6641970276832581, -0.01299323607236147, 0.5199984908103943, 0.7926396727561951, 0.02463483437895775, -1.5909770727157593, -2.502692937850952, 0.1747749149799347, -1.3379592895507812, 2.303036689758301, -0.33559128642082214, -0.6794980764389038, -0.35291624069213867, -0.48574724793434143, -1.0433722734451294, -0.6203619241714478, -1.6060841083526611, 0.5007901787757874, -0.9680317044258118, 0.8869211077690125, -0.22413277626037598, -0.7127765417098999, 1.8845090866088867, 1.2594581842422485, -0.41745176911354065, 0.20708465576171875, 0.4633035957813263, -2.197221279144287, 1.2656112909317017], [-0.9896554946899414, -0.4087415635585785, -1.0024620294570923, 1.470935583114624, 0.4919922649860382, 1.1945829391479492, 0.3349149227142334, 0.42704424262046814, 1.27646803855896, 0.5155355334281921, -0.08960255980491638, -0.9258318543434143, -0.1356203854084015, 0.7939427495002747, -2.766526460647583, -0.6502060890197754, 0.45795831084251404, -0.8241137266159058, -1.157248616218567, -2.5477986335754395, -0.7318176627159119, -0.12516936659812927, 1.8075155019760132, -0.43693146109580994, -0.422727108001709, -0.22395195066928864, 1.2821131944656372, -0.7139049172401428, -2.0037894248962402, 0.19109191000461578, -2.0727193355560303, -1.8031293153762817, -1.362480640411377, -0.4002354145050049, -1.2991782426834106, -0.5580567717552185, 0.35918113589286804, 0.6469504237174988, 0.34029266238212585, 0.6897911429405212], [-1.5955734252929688, -0.3914545476436615, 0.9518616199493408, -0.7965568900108337, 0.16995343565940857, 0.4547533690929413, 1.5790740251541138, 1.4181323051452637, -0.45660966634750366, 2.2226831912994385, -0.24843038618564606, -0.8209010362625122, 1.6064456701278687, -0.19171327352523804, 0.2527826428413391, -0.24189963936805725, -0.4207584857940674, 0.48967310786247253, 1.0344505310058594, -0.4889887571334839, 1.3315051794052124, 0.09646181017160416, -0.4462237060070038, -0.21488308906555176, 0.07791940867900848, 1.9369004964828491, -0.6668522953987122, -3.4590635299682617, -0.15719552338123322, -2.2545626163482666, 1.5993986129760742, -1.55060875415802, -0.8325281143188477, -1.1076393127441406, -1.217042088508606, 1.4832254648208618, 0.9265609979629517, -1.4142107963562012, -0.36224600672721863, 0.6025827527046204], [0.26211628317832947, -0.3301134407520294, -0.3382326364517212, 1.3014508485794067, 1.4499139785766602, 0.2700241506099701, 1.1555287837982178, 0.2985803186893463, 1.3582382202148438, 0.8918754458427429, 0.39197444915771484, 1.868330717086792, 2.4465560913085938, 1.328606128692627, -0.20463939011096954, 1.1378716230392456, -2.0319669246673584, 0.3847929835319519, 1.238725185394287, -0.5677543878555298, -1.5705009698867798, 0.6380014419555664, -0.2045588493347168, -1.0565649271011353, -1.0480577945709229, 0.3004246652126312, 1.4362585544586182, 1.8632372617721558, 0.9035146832466125, -0.061654865741729736, 0.7223149538040161, 0.540627121925354, -0.35184046626091003, 0.76899653673172, -0.787486732006073, -0.07905057072639465, 0.3556607961654663, -0.6847931146621704, 1.6066107749938965, 0.7183399796485901], [0.12432900071144104, 1.9129071235656738, -1.6511857509613037, 0.49848049879074097, 1.2957942485809326, -1.4622105360031128, -0.9988297820091248, -0.21098652482032776, 0.2710118293762207, 0.13951465487480164, 0.9087842106819153, 0.49231380224227905, 0.4277357757091522, 0.2754985988140106, -1.9008126258850098, 0.7221698760986328, -0.9020828008651733, 0.8570756316184998, -0.21136093139648438, -1.1760790348052979, 0.49778294563293457, -0.08113686740398407, 0.12132333964109421, -0.7441659569740295, -1.5589118003845215, 0.8423289656639099, -0.2901950776576996, -0.3015904426574707, -0.3157244622707367, -0.3974362909793854, 1.499375820159912, -0.2069382220506668, 1.4403398036956787, -0.592126727104187, -0.8994354009628296, -0.8480362892150879, 0.21470783650875092, 1.3564589023590088, -0.1111341044306755, 2.549598455429077], [0.09854547679424286, 1.7002540826797485, -0.24241216480731964, -0.8532799482345581, 0.2671709656715393, 1.5712755918502808, 1.645218014717102, -1.6157464981079102, 0.0419045016169548, -1.1118921041488647, -0.1046363040804863, -0.43619731068611145, 0.33178645372390747, 0.9524324536323547, -0.3161681294441223, -0.23857304453849792, -0.004353107884526253, -0.39200931787490845, 1.1687084436416626, 0.9716522097587585, 0.8546801805496216, -0.31676188111305237, 0.49888694286346436, 0.5438997149467468, -0.8477979898452759, -1.148544430732727, -1.7196992635726929, -0.12397927045822144, -0.1367674171924591, 0.33321088552474976, 0.43081364035606384, -0.5892577767372131, -2.015676259994507, 0.8656837940216064, 0.19370894134044647, 0.6583995819091797, 0.9751530289649963, -0.7717781662940979, 1.243088722229004, -1.0934360027313232], [-0.41378986835479736, 0.9464212656021118, -0.5181439518928528, -0.9316889047622681, -0.8716036677360535, 0.6431355476379395, 2.6809329986572266, -0.23009341955184937, 1.2976288795471191, -0.6245841979980469, 0.38682106137275696, 0.5158929228782654, -2.3658440113067627, -0.12931762635707855, -0.03427661582827568, -0.8010343909263611, 0.6924121379852295, -0.2814769744873047, 1.9164804220199585, -0.42340025305747986, -0.8763458728790283, -0.3356737494468689, 0.08968877047300339, 0.276161789894104, 0.7269027233123779, -0.7111122012138367, 0.5689154267311096, 1.2886037826538086, 0.7662938237190247, -2.2510313987731934, -1.4513360261917114, 1.3149538040161133, 0.7778927087783813, -2.0922904014587402, -2.6177029609680176, -0.11161036789417267, -0.8379985094070435, -0.9685145020484924, 0.5597920417785645, 0.2939036786556244], [-1.6560112237930298, 0.06911614537239075, -1.3615645170211792, -1.7119840383529663, -1.0118601322174072, 0.7204849123954773, 0.9976167678833008, 0.33938002586364746, 2.411238670349121, 1.4426989555358887, -1.8514941930770874, 0.1853671669960022, 0.6770537495613098, 1.8121461868286133, -0.2142997831106186, -1.8461834192276, -1.4396108388900757, 0.8247758746147156, 0.48160049319267273, 1.330631971359253, -0.6116132736206055, -0.9471681118011475, -0.6629621982574463, -0.44874662160873413, -0.9579263925552368, -0.5626816749572754, 0.16609430313110352, -1.1503392457962036, 0.11478543281555176, -1.625983715057373, 0.8318755626678467, 0.43135082721710205, 0.8150040507316589, 0.8480595946311951, 0.5289613008499146, -0.12184213101863861, 1.1185846328735352, 0.5024217367172241, -0.40794637799263, 0.9120901823043823], [1.1482394933700562, -0.7527099251747131, 0.5398449301719666, -0.030628010630607605, -0.47795000672340393, 1.1557468175888062, 1.2541290521621704, 0.015365568920969963, -0.04865236207842827, 0.6205994486808777, 1.5644235610961914, 0.10611164569854736, -0.8179604411125183, 1.130379557609558, -0.39721596240997314, -0.8306649327278137, -1.2180031538009644, -1.536104679107666, -0.9594883918762207, -0.11833293735980988, -1.7825573682785034, 0.07824838906526566, 1.501732349395752, -0.1510947048664093, 0.032859623432159424, -0.16842332482337952, -0.34154731035232544, 0.735836923122406, -1.624794363975525, 1.0100140571594238, -0.3492138087749481, -0.1300685852766037, 0.7237714529037476, 0.03748845309019089, -2.72686505317688, 1.127588152885437, -0.5984047651290894, 0.2642008066177368, 1.239669680595398, -0.5913575291633606], [0.8032194972038269, 1.1859922409057617, 0.6688624620437622, 0.04628414660692215, -1.5791492462158203, -0.865118145942688, 0.40645790100097656, -0.024270975962281227, -0.40297362208366394, 0.8310596942901611, 0.020316267386078835, 0.24570874869823456, -1.6095569133758545, 0.49382761120796204, 1.2521684169769287, -2.1830434799194336, 1.3302994966506958, 0.5073087215423584, 0.06505963206291199, -0.3687497079372406, 0.44962456822395325, 0.8370533585548401, -0.21321746706962585, -0.5570973753929138, 0.24847595393657684, 0.221647247672081, 2.692845106124878, -0.013340136036276817, -1.6264960765838623, -0.2263147532939911, 0.7880172729492188, 0.8376945853233337, 1.5061357021331787, -0.2331072986125946, 1.5200440883636475, 0.002600704552605748, -0.49956396222114563, -1.2238706350326538, 0.2128164917230606, 0.29785504937171936], [-1.5921920537948608, -0.33907392621040344, 2.1043989658355713, 0.913214921951294, 1.2982254028320312, -0.5162732005119324, -0.9817183017730713, 0.5186852812767029, -0.9685202240943909, -0.20625275373458862, 0.5995126366615295, 1.2143793106079102, 0.9973539710044861, -0.11299443244934082, 0.3599962592124939, 0.5379982590675354, 0.03799906745553017, 0.6819029450416565, 0.589440643787384, 0.5948742032051086, 1.3095459938049316, 1.1329704523086548, 0.7310620546340942, 0.2512543797492981, -1.5230392217636108, 1.0394196510314941, -0.8959292769432068, -0.6960854530334473, 1.7496517896652222, -1.4589298963546753, -0.1221892386674881, 0.32900696992874146, -0.7070080041885376, -2.621513843536377, -0.2207949310541153, -1.0564849376678467, 0.5085407495498657, -0.1270310878753662, -0.1586865782737732, 1.246994137763977], [-1.3594330549240112, -0.6638349890708923, 0.40026673674583435, 0.5121188759803772, 0.09531795233488083, 0.7813583016395569, 0.8423083424568176, -0.5816363096237183, 0.4879056513309479, -0.6701844334602356, -0.594086229801178, 0.4005949795246124, 1.3625856637954712, 0.0006735569913871586, -0.5247341394424438, 0.13124190270900726, -1.637055516242981, -0.7511032223701477, -1.159091830253601, -0.4232158064842224, -0.8092561364173889, 0.8232645392417908, 1.7032142877578735, 0.08962372690439224, -0.15625667572021484, -1.2410675287246704, -1.1935282945632935, 0.8438053727149963, 0.3820584714412689, -1.1762721538543701, -1.523547887802124, 0.5022447109222412, -2.586322546005249, -0.9975312352180481, -0.09056936949491501, 0.35369428992271423, 1.3199435472488403, -1.2238317728042603, -0.4134069085121155, -1.010384440422058], [0.17525003850460052, 0.2531420886516571, -0.028352349996566772, 0.5100297331809998, -0.1348164677619934, 0.9785096645355225, 0.859642744064331, 1.0959471464157104, 1.3468443155288696, 0.3145889937877655, 1.1295078992843628, -0.16191425919532776, 0.9683593511581421, -0.3763748109340668, -0.9322926998138428, 1.5790480375289917, -0.20576618611812592, -1.0164375305175781, 1.6070868968963623, 0.0331282801926136, -0.9001510143280029, 1.365882396697998, 0.3788755536079407, 2.597621440887451, 1.0998045206069946, 0.5863693952560425, 0.5258246660232544, 0.03678412735462189, -0.18735966086387634, -0.3287767171859741, 1.3442916870117188, -0.5000052452087402, 0.20057156682014465, -0.05909798666834831, -1.398804783821106, -0.01670764945447445, -0.19640770554542542, -2.706923484802246, 1.6081328392028809, -0.039802853018045425], [1.6258749961853027, -0.04220104590058327, -0.569811999797821, -0.83833247423172, -0.6489275693893433, 0.018199222162365913, -1.621846318244934, -0.20470450818538666, 0.9405168294906616, 0.04315866902470589, 1.1269733905792236, -0.5671367049217224, 1.1352087259292603, 0.6975420117378235, 3.317173957824707, -0.532439649105072, 1.8700604438781738, 0.7818974852561951, -1.533473253250122, -0.5642361044883728, -0.6573771238327026, -0.9622738361358643, 0.17621366679668427, 0.4262404143810272, 0.6934133768081665, 0.8929754495620728, -0.7490493655204773, -0.49467575550079346, 0.07539328187704086, -1.4652220010757446, -0.30960050225257874, 0.029621481895446777, 0.14790506660938263, 0.8006096482276917, -1.7941502332687378, -0.4318939447402954, 0.032770175486803055, -1.1520479917526245, 0.3690095841884613, 0.5559400320053101], [-1.5407341718673706, 2.516894578933716, 0.4587540328502655, -0.49022331833839417, -2.227168083190918, 1.5863033533096313, -0.08475352078676224, 0.5602328181266785, 0.7742997407913208, 0.9826062917709351, -0.18266156315803528, 1.438645362854004, 0.7975227236747742, 1.2097400426864624, 0.25547873973846436, 0.8969470858573914, -0.6503251194953918, -1.7535797357559204, -1.084600567817688, 0.602388322353363, 0.08706340193748474, -2.462616205215454, -0.5044109225273132, 0.1337888389825821, 0.14197301864624023, 1.3110907077789307, -1.5105414390563965, 0.6116389036178589, 1.3322805166244507, -0.04080303758382797, 1.0830297470092773, -0.32358887791633606, -0.13031436502933502, 0.577698290348053, 2.3414900302886963, -0.24185484647750854, -0.6580716371536255, 0.05324629694223404, 0.521099865436554, -1.1199722290039062], [-0.8429291844367981, 1.1421735286712646, 0.01833294704556465, 0.2923007011413574, 0.5507246851921082, -0.26250797510147095, 0.4759361743927002, 0.46540141105651855, 0.5775597095489502, -0.7056830525398254, -0.6928060054779053, 0.537293016910553, 0.597740113735199, -1.1828742027282715, -0.4348874092102051, 0.898800790309906, 1.259000539779663, -0.921238899230957, 0.6957346200942993, -0.2788313031196594, 0.9142029285430908, 0.9139748811721802, 0.7208530306816101, 1.3769210577011108, -2.03728985786438, -0.9688913226127625, 0.5000783801078796, -0.17570403218269348, -0.6801699995994568, -0.5827474594116211, -0.6167094111442566, 0.2541826069355011, 0.38666653633117676, 0.45424172282218933, -0.9941412210464478, -0.08302600681781769, -1.013289213180542, 0.43439146876335144, 0.5138027667999268, 0.07024529576301575], [-0.24310322105884552, 1.354488730430603, 0.36280491948127747, -0.9594920873641968, 0.5240615606307983, 0.6798611283302307, 0.3848743438720703, 1.3898992538452148, -0.5302911400794983, -0.5295931100845337, -0.4134718179702759, 0.16011971235275269, -0.21826381981372833, -0.2525043785572052, 0.6581162810325623, -0.20118482410907745, 0.2523242235183716, -0.13693320751190186, -0.9903852343559265, -1.7607834339141846, 1.5108474493026733, -2.0290768146514893, 0.7651028037071228, -0.2041427493095398, -0.5007688403129578, 1.2772175073623657, -0.6085683107376099, -1.8433955907821655, 1.9978091716766357, -0.49619877338409424, 0.048342570662498474, 0.46608254313468933, 0.41559353470802307, -0.8358225226402283, -0.1156844049692154, -0.09501901268959045, 0.881607174873352, 0.46503525972366333, -1.2600867748260498, -0.7181721329689026], [1.1365858316421509, 0.7143035531044006, 0.9741303324699402, -1.6079386472702026, -0.2662545442581177, -1.1089786291122437, -0.38917672634124756, -0.7215077877044678, 0.08051824569702148, 0.25585436820983887, -1.1905287504196167, 0.8858138918876648, -0.2605067491531372, -1.6358587741851807, -0.506655752658844, -0.31427720189094543, -0.10106267035007477, -0.6508522033691406, 0.29065394401550293, 0.48223167657852173, -0.5453011989593506, 0.7947776317596436, -0.19427354633808136, -0.8645853996276855, -0.7587139010429382, -0.6831890344619751, -0.18204428255558014, -0.4583171308040619, 0.707478404045105, 0.7178645730018616, 0.03621487691998482, -0.9630021452903748, 0.13561522960662842, 0.2742665410041809, -0.18092790246009827, -0.65987229347229, 0.8635284304618835, 1.351547360420227, -0.05151322856545448, -0.3186362087726593], [0.019140707328915596, -0.6664561033248901, 0.6675089001655579, 0.1601872444152832, -0.12965047359466553, -1.3945198059082031, 0.3472517132759094, -0.7068962454795837, 0.017391838133335114, -0.38643693923950195, 0.3411429822444916, 0.9871745109558105, -0.6217589974403381, 1.6799639463424683, 0.14584480226039886, -1.4174818992614746, 1.3298637866973877, 0.8137038350105286, -2.2692947387695312, -0.5231112241744995, -0.10112747550010681, 1.172009825706482, 0.18543468415737152, -0.64262855052948, 0.22462716698646545, -0.250105082988739, 1.3485298156738281, 1.324671983718872, -0.2782258689403534, 1.0294897556304932, -0.2819913923740387, 0.6286312341690063, -1.5203852653503418, 0.3239454925060272, -2.58856463432312, 1.0793569087982178, -0.7433406710624695, -0.5790044069290161, -0.02838818170130253, -1.0314903259277344], [-1.12405526638031, 0.6073639988899231, 0.5544313788414001, -2.809079170227051, -0.6302061080932617, -0.09763482958078384, -0.6192594766616821, 0.65391606092453, 0.4983258843421936, 0.16873857378959656, 1.2128695249557495, 0.2087235450744629, 1.6906795501708984, -1.6271569728851318, 0.6852872967720032, 0.04041994735598564, -2.5013844966888428, 1.4399526119232178, -0.6683886051177979, 3.051609516143799, 0.703761637210846, 0.30385157465934753, -2.117619037628174, -0.38435208797454834, -1.8294415473937988, -1.3010857105255127, -0.41861265897750854, 0.8742368221282959, -0.12279666215181351, -2.231567621231079, -1.2780344486236572, 0.5944381356239319, 1.7513288259506226, 1.0458046197891235, -0.561947226524353, -0.43675193190574646, 0.1509818285703659, -0.8235414624214172, 1.1116100549697876, 0.9444110989570618], [0.9971170425415039, -0.716518223285675, 0.5732001662254333, -0.7477449178695679, -2.2103915214538574, 0.23637229204177856, -0.8301711082458496, -0.9063159823417664, 0.18673309683799744, -1.412882685661316, -1.6495797634124756, -0.266048401594162, 0.09770943969488144, -2.1389782428741455, -0.256673127412796, -0.3313136696815491, 0.0025273405481129885, -0.5915681719779968, -0.7732288837432861, 0.9035683274269104, -0.1247807964682579, 0.5969070196151733, -1.445630431175232, -1.7495274543762207, -0.05876133218407631, 0.28399041295051575, -0.3254416584968567, -0.3086971938610077, 0.3485957384109497, -2.1051342487335205, -0.39301377534866333, -0.8168322443962097, -0.3812505304813385, -0.7367311120033264, -0.17788858711719513, -0.209361732006073, 1.5319101810455322, 1.213166356086731, -2.3660855293273926, 1.6236997842788696], [0.38675424456596375, 1.7670799493789673, 1.154958963394165, 0.6367597579956055, -0.1291675716638565, 1.1470683813095093, 1.1373628377914429, -0.8474756479263306, -0.5351696014404297, 0.4838179051876068, -2.864215850830078, 0.16027027368545532, -0.6410157680511475, 2.1470417976379395, 0.009893235750496387, -0.5430402755737305, 0.21091076731681824, -0.9149774312973022, 0.4776811897754669, -0.27139919996261597, 0.004396030679345131, 0.9898141622543335, -0.7757821679115295, 0.1057383269071579, 1.0081194639205933, 1.8465806245803833, 0.2647460699081421, 0.33877962827682495, -0.05489716678857803, 0.33672890067100525, -0.04542010277509689, 0.7177976965904236, 1.435158610343933, -0.10193010419607162, -1.1218738555908203, -0.6601489186286926, -0.3485061526298523, 1.3144237995147705, 0.41193485260009766, -0.5624481439590454], [0.6348034739494324, -0.43634703755378723, -0.5551295876502991, 1.9195220470428467, 0.24828164279460907, 0.5250255465507507, 0.2803367078304291, 0.21704722940921783, 1.0234415531158447, 0.6403588056564331, 1.0328807830810547, -0.3121528625488281, 0.11331083625555038, 1.22230863571167, -0.5421299338340759, 0.427755743265152, 1.5201711654663086, 0.27142250537872314, -2.4624524116516113, -0.7097930312156677, -1.074387788772583, -0.2519668638706207, 1.1855201721191406, 0.29352983832359314, -1.9454400539398193, -0.41616564989089966, -0.20789310336112976, -0.24292291700839996, -0.031429022550582886, 0.5916496515274048, 0.9889854788780212, -0.15179485082626343, -0.5833149552345276, 0.57989102602005, -0.36287447810173035, -0.08836333453655243, 0.3850562274456024, -0.6274950504302979, 0.5154465436935425, 0.8596118688583374], [1.1737629175186157, -0.2908913791179657, 0.5657860636711121, -1.1736873388290405, -0.11872026324272156, -0.3389348089694977, -1.2747184038162231, -0.5109438300132751, 0.9680270552635193, 0.02492438070476055, -0.13083647191524506, 0.816744863986969, 0.6171470284461975, -0.9141494035720825, 1.4462411403656006, 0.030091747641563416, 0.9222784042358398, -0.2745337188243866, 0.8195526599884033, -1.9113670587539673, -1.2901197671890259, -0.5769140124320984, -2.0922415256500244, -0.6217818260192871, 1.3234306573867798, 1.9097572565078735, -0.30354127287864685, 0.11957371979951859, -1.4238840341567993, -0.747607409954071, 0.2835409343242645, -1.028532862663269, -1.1121060848236084, -0.1091267392039299, 1.6933939456939697, 1.090521216392517, 0.26662659645080566, 2.4579460620880127, 0.259063720703125, -0.9654499888420105], [-0.48427748680114746, 1.2347004413604736, 0.7627307772636414, -0.4018230736255646, -0.6110784411430359, 0.2942279577255249, 0.061038143932819366, 0.6084339618682861, 0.4562288820743561, 0.30698925256729126, -0.624904215335846, 0.012022474780678749, -0.9093824028968811, 0.16950812935829163, -0.3009333908557892, 1.0798845291137695, -0.13096772134304047, 0.1800038069486618, -0.22374321520328522, 0.056514423340559006, -0.37047556042671204, -2.1375322341918945, -0.20296692848205566, -0.32155320048332214, -0.2570927143096924, -0.4462636709213257, 3.5936641693115234, 0.5388694405555725, -1.7884479761123657, -0.37268373370170593, -1.2400604486465454, -0.5135223865509033, -0.10837025195360184, -0.5471975803375244, -0.14000321924686432, -1.6974084377288818, -0.941677987575531, -0.12432608753442764, 0.39510640501976013, 0.8992461562156677], [-1.2980369329452515, 0.17079277336597443, -0.0143435662612319, -1.161920189857483, 0.9460285305976868, -1.1246541738510132, -1.0653800964355469, -0.36285877227783203, 0.4169304072856903, 1.9583122730255127, 0.49010929465293884, -0.7409818768501282, 0.010563580319285393, -1.4288082122802734, -0.04400329664349556, -0.02708737552165985, -1.6032359600067139, 0.7983970046043396, 0.7091252207756042, -1.3705997467041016, -1.3818870782852173, 0.1587253212928772, 0.14065399765968323, -1.095733404159546, 1.1530874967575073, 0.04688321799039841, 0.5866249799728394, 0.6305869221687317, 0.8802684545516968, -0.142785906791687, 0.34323030710220337, -0.7643119692802429, -1.1672792434692383, -0.4361339211463928, 0.40691983699798584, 1.1361175775527954, 1.0419483184814453, -0.09461218863725662, -1.0237560272216797, -0.4277247488498688], [0.7160154581069946, 1.440269112586975, -0.481139600276947, 1.2942161560058594, -0.6244021058082581, -0.15289713442325592, -0.6226464509963989, 0.8798981308937073, -1.2671136856079102, 1.0668256282806396, -1.781111240386963, -0.11623988300561905, 0.5318050980567932, -0.10339837521314621, -0.6876583099365234, -0.5084636807441711, 1.6024370193481445, -0.06702198088169098, 0.8777444958686829, 1.9222102165222168, 1.4756051301956177, -1.3641451597213745, -1.2493449449539185, 0.45869460701942444, -0.004084739368408918, -0.21336692571640015, 1.349544882774353, -0.3103613257408142, -0.8082079887390137, 0.6985368728637695, 1.0454413890838623, -0.004955647978931665, -1.1603885889053345, -0.775002658367157, 1.7238587141036987, 1.070386290550232, -0.146625354886055, -0.3856590986251831, -0.17497652769088745, -0.7998202443122864], [0.03833794221282005, -1.5800701379776, -1.0468295812606812, -0.3356373608112335, -0.8364565372467041, -0.7845628261566162, 1.0321885347366333, 0.3119436502456665, -0.12810826301574707, -0.9180323481559753, -0.26197269558906555, 0.11738656461238861, 1.3471463918685913, 1.1399012804031372, 0.1598966270685196, -1.0602765083312988, -1.5021815299987793, -0.5392816066741943, 0.779346764087677, -0.7475239038467407, -0.25978320837020874, 0.5732178092002869, 2.098597288131714, -0.4799070358276367, -0.16840475797653198, -0.06597857177257538, -0.49845466017723083, 0.6846109628677368, -0.9495740532875061, 0.8494457006454468, -0.48523929715156555, 0.9918031692504883, -0.7569202780723572, 0.8685823678970337, 0.15498223900794983, 1.350808024406433, 1.0590839385986328, -0.928602397441864, 0.6965193748474121, 0.7393049597740173], [0.011350140906870365, -0.4244448244571686, -0.8391701579093933, 1.7481410503387451, 0.5048165917396545, 0.4829320013523102, -1.7922478914260864, -0.4873296916484833, 0.25373128056526184, -1.7062950134277344, 0.07789302617311478, 0.15352435410022736, 0.3451903760433197, -0.7713280320167542, 1.8571282625198364, -0.9345723390579224, -0.6129924058914185, -0.7149192690849304, -0.42800262570381165, 1.3887165784835815, 0.4274701476097107, 1.6934611797332764, 1.0714020729064941, 0.3151838183403015, 2.0806915760040283, -0.6760984659194946, 0.6304818391799927, -1.203805923461914, 0.041232381016016006, 0.772664487361908, 0.7090508937835693, 1.6708091497421265, -0.3404635488986969, 1.9410059452056885, 0.768018364906311, 0.5678909420967102, -0.38351231813430786, -0.8624584674835205, -0.5348954200744629, 1.002092957496643], [-0.714134156703949, 0.6307637691497803, 1.3462989330291748, -0.7449103593826294, -0.4149552583694458, 1.2105096578598022, 0.10332482308149338, 2.1326234340667725, 0.3678067922592163, 1.1053496599197388, -0.49136456847190857, -1.322235345840454, 0.1319780945777893, -2.6596438884735107, -1.6346206665039062, -1.3096776008605957, 0.602766752243042, 0.29819443821907043, -1.043980598449707, -1.2911090850830078, -0.014485875144600868, -0.2503599524497986, 1.827772617340088, -1.131837248802185, -0.3188209533691406, -0.2383188009262085, -0.32337629795074463, -0.20813414454460144, -0.9158944487571716, -1.4715783596038818, -1.8494757413864136, -0.4902924597263336, -0.2539251446723938, -1.002206563949585, -0.7021263837814331, 1.4472887516021729, 1.8749161958694458, 1.2079073190689087, -0.8567982912063599, -0.1344219446182251], [-0.03127876669168472, -0.4526667296886444, -0.3824440538883209, -2.710066318511963, 1.4615974426269531, 0.22220070660114288, -0.019337758421897888, 0.43319228291511536, -1.0111706256866455, 0.5350334048271179, -0.09012498706579208, 0.04135711118578911, -0.9794788360595703, -0.1465299427509308, 0.01070867944508791, 0.2389003187417984, -1.3505572080612183, -0.07311961054801941, 0.4060736298561096, -0.05544080585241318, -1.6495198011398315, 1.5412333011627197, 0.18034185469150543, 0.43909329175949097, 0.1542230248451233, 0.5213225483894348, 0.6430844068527222, 1.7527306079864502, -0.980838418006897, -1.7019132375717163, -0.5934287309646606, -2.2490222454071045, 2.3381075859069824, -0.43730586767196655, -0.6713283658027649, 0.013566844165325165, 0.9176728129386902, -0.8590949773788452, -1.5644493103027344, -1.8460781574249268], [0.36667871475219727, 0.5923285484313965, -0.41367223858833313, 1.8908307552337646, 0.011302091181278229, -1.7199418544769287, 2.1333680152893066, -1.142911672592163, -0.24913184344768524, -0.2778846025466919, 0.6358503103256226, 1.538374423980713, -0.4659469425678253, 0.9688255786895752, 0.029547682031989098, -0.10135196894407272, 0.39364010095596313, -0.5860187411308289, -0.21520088613033295, 0.9533432722091675, -0.43625420331954956, -1.1783307790756226, 0.07275322079658508, -1.465585708618164, -0.17808936536312103, -2.0712802410125732, -0.16490817070007324, 0.8894903659820557, 2.025334119796753, -0.8420898914337158, 0.8114293813705444, -1.2003253698349, -0.8745846152305603, -0.11328853666782379, 0.11602392792701721, -0.1861342042684555, 0.5053884387016296, 1.3922474384307861, 0.7160592675209045, 0.37426215410232544], [0.21841001510620117, 0.1690983921289444, 0.49209001660346985, -0.31149232387542725, -0.48052507638931274, -0.10227754712104797, -1.4015491008758545, 1.0336757898330688, 0.3612714111804962, 0.04201279580593109, -0.0667252317070961, -0.6437427997589111, -0.7728284597396851, 2.289003372192383, 0.11887511610984802, 1.2488758563995361, 0.5923484563827515, 1.7772694826126099, 0.3293660879135132, -0.8198394775390625, 2.0529861450195312, -0.06986258178949356, 1.1496318578720093, -1.1338971853256226, -0.6838953495025635, -0.596032977104187, 0.8105660080909729, -0.7583895921707153, -0.29154399037361145, 0.19777363538742065, 0.21797871589660645, -2.9605350494384766, -0.06924063712358475, 0.4109583795070648, 1.6601264476776123, 1.4525396823883057, -0.3919629752635956, -0.6652529239654541, 1.4095112085342407, 0.5490295886993408], [0.4426451325416565, -0.4482896029949188, -1.3780829906463623, -0.01776622235774994, 0.840394139289856, 0.3016184866428375, -0.4369645118713379, 0.8595510125160217, -0.5109685659408569, -1.6025053262710571, -1.7255022525787354, 0.12685801088809967, -0.5741806030273438, -0.37755611538887024, -0.22296647727489471, -0.10739339143037796, 0.4410606026649475, 1.433188557624817, 1.4012826681137085, -1.0771244764328003, 0.9067248702049255, 1.002713680267334, -1.50246000289917, 0.3422122895717621, 1.948576807975769, -0.17131009697914124, 0.30886632204055786, 0.47714880108833313, -0.36812156438827515, -2.7229526042938232, -1.142855167388916, 1.1991827487945557, -0.9416080117225647, -0.8354716300964355, -2.263139486312866, -1.0634640455245972, -0.1284545511007309, 0.7512815594673157, -2.8477725982666016, -0.6000233888626099], [1.55997633934021, 0.5367847681045532, 1.0214662551879883, -2.1662235260009766, -0.9012369513511658, -0.46765458583831787, -0.7750309109687805, 0.38050732016563416, 0.8014460206031799, -0.1519167274236679, 0.13907283544540405, 0.1402668058872223, 0.44796061515808105, 0.6063187122344971, -0.0032724542543292046, -1.2304153442382812, 0.7810744047164917, -1.2973051071166992, 0.5870575904846191, 2.2659919261932373, 0.07638455182313919, -1.0928597450256348, -0.40306517481803894, 1.6777565479278564, -0.5018728971481323, -0.4342029094696045, 0.9169792532920837, 0.7689864039421082, -1.290350317955017, 2.783693552017212, -0.5242080688476562, 1.2037242650985718, -0.48825180530548096, 0.5655708312988281, -0.5151016116142273, 0.20951446890830994, 0.2686864137649536, -0.5540139079093933, 0.6130457520484924, 0.7076656818389893], [0.49949562549591064, 0.40500104427337646, -0.6995679140090942, -1.1162235736846924, -2.2628684043884277, -1.047189712524414, -0.18437205255031586, 0.7653350830078125, -1.4455065727233887, 0.2462356835603714, -0.4868408739566803, -1.5352888107299805, -0.5851866006851196, -0.9257246851921082, 2.019070625305176, 0.32193902134895325, -0.5977641344070435, -1.3046495914459229, -0.4814511239528656, 1.0756406784057617, -0.6289405226707458, 1.0984464883804321, 0.2598603367805481, -0.4031612277030945, -0.5594173669815063, -0.09991873055696487, -1.0321687459945679, -0.6661319732666016, 0.01923968456685543, 0.9414905309677124, 1.8844109773635864, -0.3577636480331421, -0.5669267773628235, 0.014056256040930748, 0.8112053871154785, 1.5759252309799194, -0.6891899704933167, -0.5821748375892639, -0.7644881010055542, -0.2501354217529297], [-1.3230983018875122, 1.4790122509002686, -1.5854954719543457, 2.592966318130493, 1.239506721496582, 0.5267500877380371, 1.5164905786514282, -1.0745550394058228, -2.812298536300659, 0.3868802785873413, -3.382513999938965, -0.5075410008430481, -0.15167725086212158, 0.6221721768379211, 0.9241339564323425, 0.4910140931606293, 0.16107837855815887, -1.3854516744613647, -0.004747070837765932, 0.17502905428409576, -0.7045135498046875, -1.489467740058899, -0.40542545914649963, -0.5185328722000122, -2.041285991668701, 1.4242311716079712, -0.6057809591293335, 0.7240262031555176, 1.4349610805511475, 1.02881920337677, 2.1253111362457275, -0.7822997570037842, -1.6962567567825317, 1.1601701974868774, -1.067627191543579, -0.41854050755500793, -0.48022255301475525, -0.661041796207428, -0.025221016258001328, -0.27900442481040955], [-1.0458323955535889, -1.077829122543335, -0.14286506175994873, 1.0811519622802734, -1.2884531021118164, 0.9458540081977844, -0.6141991019248962, 1.2258659601211548, 0.3029814064502716, -0.04571431502699852, -0.034750841557979584, -1.6481387615203857, -1.7920491695404053, -1.2751330137252808, 0.9775420427322388, 0.9447106719017029, 0.7135264277458191, 1.844908595085144, -1.1351584196090698, 1.427296757698059, -0.4104268252849579, 0.11853813380002975, 1.5862773656845093, 0.26256081461906433, -0.629993736743927, 0.10731913894414902, 0.8791733384132385, -0.13971315324306488, -1.1175976991653442, -0.0525817833840847, -0.969365656375885, -0.6054254770278931, -0.9820910692214966, 0.2465508133172989, -1.122660517692566, -2.01660418510437, 0.6055225729942322, -0.2711279094219208, 0.37724170088768005, -0.17666788399219513], [0.3330194056034088, 0.26502227783203125, 1.0520219802856445, 1.932943344116211, 0.6224358677864075, -1.4806307554244995, 0.8004539608955383, -0.6933348774909973, 0.15136493742465973, 0.33113008737564087, 0.08850840479135513, -2.2118165493011475, 0.3836519122123718, 1.0290019512176514, 0.2838759124279022, 0.3174342215061188, 0.7811059951782227, 1.2152739763259888, -0.21397024393081665, 0.5662411451339722, 0.7500925660133362, -0.09806077182292938, -0.5135445594787598, -0.7101869583129883, 0.09632588177919388, 0.4496254622936249, 0.7050826549530029, -0.15495766699314117, 0.5722958445549011, -0.10341382771730423, -0.517103374004364, -0.15464149415493011, 0.674394965171814, -1.047609806060791, 0.5260472297668457, 1.0653953552246094, -1.6647701263427734, 0.5577123165130615, 0.03382217139005661, 0.06553643941879272], [-1.6757570505142212, 2.1758313179016113, -1.080852746963501, 0.7034095525741577, -0.7751353979110718, -0.5191704630851746, 2.4602835178375244, -0.5620823502540588, -0.5645849704742432, -0.037586554884910583, 2.251544952392578, -0.89383465051651, 1.6592658758163452, -0.5801072716712952, -0.088550865650177, 0.30124667286872864, 0.04350517317652702, -0.048093281686306, 1.473902702331543, -2.237194538116455, -0.09630695730447769, -0.5909241437911987, 1.331884741783142, -0.28259554505348206, -0.7783903479576111, 0.2387302964925766, 0.34843891859054565, 1.1253496408462524, -1.7882416248321533, 0.03726795315742493, -0.15976843237876892, 0.7611641883850098, 0.5851231813430786, -0.05541924387216568, 1.2569416761398315, -0.11487147957086563, -2.552661657333374, 0.7293055057525635, -0.8160033822059631, -0.4699923098087311], [-1.0568089485168457, -0.8308737277984619, 0.6856114268302917, 0.9808275103569031, 0.1271030604839325, -0.42882972955703735, 1.3288935422897339, 0.9245702624320984, -0.4029918611049652, -0.9679842591285706, -0.23387277126312256, -1.5153154134750366, -0.5938131809234619, 0.24322468042373657, -0.4297211766242981, 0.5159046053886414, 1.0989124774932861, 0.010966125875711441, 0.15802672505378723, 1.4240496158599854, 0.9263299703598022, 1.5142531394958496, -1.8314884901046753, 1.6221576929092407, -0.1842198520898819, -0.1572798639535904, -0.30024540424346924, -0.23939965665340424, -0.5275037288665771, 0.11380703002214432, 0.7167568802833557, -1.078237771987915, -1.6894359588623047, 0.8924429416656494, 1.8102928400039673, 0.40143874287605286, -0.0262460857629776, 0.8286005854606628, -1.6952887773513794, -0.5363584160804749], [0.14279787242412567, -0.6626290082931519, -1.9414761066436768, 0.029852109029889107, -1.7387049198150635, 0.20375487208366394, -1.294797420501709, -1.7039928436279297, 0.2936991751194, -0.2299758791923523, 0.9810386300086975, -0.020397044718265533, -0.3463212549686432, 1.000413179397583, -1.134535551071167, -0.5289199352264404, 0.4281543791294098, 1.1936604976654053, 0.4697701036930084, 1.1061166524887085, -0.2780417203903198, 0.9863590002059937, 0.5992802977561951, -0.7443291544914246, 0.9055185317993164, 1.3028509616851807, 1.4006074666976929, 1.7261149883270264, -0.6150985360145569, -1.4949520826339722, -1.1566954851150513, 1.2120676040649414, -0.11518754065036774, 1.319300889968872, -1.430086374282837, -0.4353977143764496, 0.7805774807929993, 1.3597817420959473, 1.3581358194351196, 0.5551024675369263], [-1.2739055156707764, -0.8951661586761475, -0.6055532693862915, -0.8188033699989319, 0.04192307963967323, -0.0019860456231981516, -2.010465145111084, -1.0180480480194092, 1.5649573802947998, -2.0211894512176514, -0.33973607420921326, -0.35805264115333557, -0.3307635486125946, 1.148829698562622, 0.9119971990585327, -0.2506769001483917, -0.015971850603818893, -0.18013499677181244, 0.8565429449081421, -0.4913327395915985, -1.5378018617630005, 1.8786321878433228, -1.0107494592666626, -0.15369117259979248, 0.466789186000824, -0.2401452213525772, 0.2128371000289917, -0.5494675636291504, -0.4352550208568573, 0.7683509588241577, -0.28810569643974304, 0.27473703026771545, 1.5067570209503174, -0.7086883187294006, 0.3568818271160126, 0.5388199090957642, 0.26006630063056946, 0.702849805355072, 0.32232901453971863, -0.4799722135066986], [1.5887930393218994, -1.9703339338302612, -1.1146979331970215, 2.180959701538086, -0.7076213955879211, 0.5331335663795471, -0.690464437007904, 1.2547721862792969, 0.4034481644630432, -0.22908984124660492, 0.5956401228904724, 0.02316674403846264, -0.6243965029716492, 1.4536845684051514, 1.5056724548339844, -1.1912332773208618, -0.7308636903762817, 0.912053108215332, 0.6761382818222046, -0.5299085378646851, -0.14903907477855682, 0.105274997651577, 0.32904401421546936, 0.6308162212371826, 0.21430714428424835, 0.2512320876121521, -0.726537823677063, 0.7695218324661255, -0.9200353026390076, -0.3732832074165344, 1.9000107049942017, -0.7303234934806824, 0.2906946837902069, -0.5467517971992493, 0.847281813621521, -1.3078460693359375, 1.0025112628936768, -1.5036014318466187, -1.2986807823181152, -0.8449689149856567], [0.18475548923015594, 3.0706686973571777, -0.19803503155708313, -0.22421376407146454, -0.09475359320640564, 0.06353825330734253, 0.13129016757011414, 0.2911517024040222, -1.2728224992752075, 0.4830520749092102, 0.4417300224304199, 0.29115933179855347, 0.28740450739860535, 1.483573317527771, -0.8737273812294006, 1.5340018272399902, -0.13147103786468506, -0.5382584929466248, -0.7253328561782837, -1.106595754623413, 2.0847840309143066, 0.9828638434410095, 0.5742500424385071, 0.2923673391342163, -1.6052335500717163, 0.08654797077178955, -0.8393992185592651, 0.5572416186332703, 1.8275877237319946, -0.5902314782142639, -1.225258708000183, -0.5397655963897705, 0.748458743095398, 0.12027356028556824, -0.6212554574012756, -0.8558071851730347, 1.832215666770935, -0.6800710558891296, 2.41506028175354, 1.3837627172470093], [0.36693471670150757, -0.29390108585357666, 1.975932002067566, -1.7927836179733276, -0.4447946846485138, -0.06896192580461502, 0.5190581679344177, -0.24383509159088135, 0.5061551928520203, -0.33184707164764404, 0.4296136498451233, 1.518361210823059, -0.8118845820426941, 1.3304208517074585, 0.9538536071777344, 1.2253632545471191, -1.8136250972747803, -0.48302972316741943, -0.18856368958950043, -0.05399147421121597, -0.1204606145620346, -1.1396689414978027, -1.0955427885055542, 0.5090059041976929, 1.1260238885879517, 1.5562328100204468, -1.2789679765701294, -1.4360562562942505, -1.3077064752578735, 1.3541138172149658, -0.7401880025863647, -0.6930949091911316, -1.8647148609161377, 0.5997684597969055, -0.17802482843399048, 1.3029000759124756, -1.5902683734893799, 0.051617346704006195, 0.1018173024058342, -1.2711235284805298], [-0.1287880837917328, 0.4951169788837433, -0.4974000155925751, 0.5863596200942993, -0.010556288994848728, 0.3405821621417999, 1.3852328062057495, 1.2796506881713867, 0.511544406414032, 0.9170652031898499, -1.1766788959503174, 0.05942264199256897, 1.047887921333313, -0.8476803302764893, -2.1900382041931152, 0.5091376900672913, -0.5991914868354797, 0.15922532975673676, -2.0956621170043945, 1.1043201684951782, -0.9883697032928467, -0.6023745536804199, 0.7276758551597595, 0.9915555715560913, -0.48534512519836426, -1.7816065549850464, 0.8384366035461426, -2.666818618774414, -0.2555270493030548, 2.3787055015563965, -0.496793270111084, 0.072264164686203, 0.977853000164032, -0.6215537786483765, -0.5618536472320557, 1.2945811748504639, 0.537246584892273, 0.4398523271083832, -0.7635974287986755, -0.24016763269901276], [1.1790531873703003, 0.6724739670753479, 0.28853824734687805, 1.2574139833450317, 1.1659471988677979, 1.6753442287445068, 0.766817033290863, -1.1031769514083862, -0.2137180119752884, -1.0329111814498901, -0.16851043701171875, 0.14604170620441437, -0.547076404094696, -2.535252094268799, 0.169255331158638, -1.2531250715255737, 0.8308126330375671, -1.748507022857666, 0.8362657427787781, -0.41767069697380066, 0.027518734335899353, 0.9781255722045898, -1.760939121246338, -0.18588675558567047, -0.07831713557243347, 0.8675766587257385, 1.267683744430542, -0.7517688274383545, -0.1964424103498459, -2.0860280990600586, -0.42562568187713623, 0.15818245708942413, 0.8158868551254272, 0.7401086688041687, 0.7177376747131348, 0.1549769639968872, -0.3967263698577881, 0.7457531690597534, -0.476387083530426, 1.148292899131775], [0.9978660941123962, 1.4774097204208374, -0.9144805073738098, -0.44264546036720276, -0.9613029956817627, -0.20013999938964844, 0.8520175814628601, -1.1076114177703857, -0.37864235043525696, 0.09243080765008926, 0.47949644923210144, -0.1642269790172577, 0.09879696369171143, -1.5726526975631714, -0.3214479088783264, -0.2154112607240677, -0.3556956946849823, 0.04513269662857056, 1.8038733005523682, -1.727679967880249, 0.65400230884552, 0.18692266941070557, -0.7804679870605469, -0.07331668585538864, 0.35009750723838806, 0.5633726119995117, -1.297304391860962, -1.3542046546936035, -0.22907084226608276, -0.033510368317365646, 0.1405017375946045, 0.10806320607662201, -1.3130927085876465, 0.08268935233354568, -0.3630063533782959, 1.2540127038955688, 0.0641990453004837, -0.33985045552253723, -1.671065330505371, -0.4429291784763336], [-0.0781196728348732, -0.48387598991394043, 0.4139997959136963, 1.15570867061615, 1.8475396633148193, 1.4907983541488647, 0.540747880935669, -1.0677233934402466, -0.9477317333221436, -0.11456003785133362, 1.1735460758209229, -0.27044177055358887, 1.2074098587036133, 0.8747744560241699, 2.879499912261963, 0.049341924488544464, -2.6425211429595947, 1.6531012058258057, -0.27891987562179565, -1.3707903623580933, -0.06980016082525253, 1.7803573608398438, 1.6941279172897339, -0.020726654678583145, -1.4898864030838013, -0.523859977722168, 0.3773260712623596, 0.8650361895561218, 0.351220041513443, -0.2894684076309204, 1.1291453838348389, -0.5737378597259521, -0.15079721808433533, -0.6581683158874512, -0.4385198950767517, 2.4974381923675537, 0.7781424522399902, 0.9068387746810913, -0.3709871768951416, 0.3407028913497925], [-1.476378321647644, -0.10408017039299011, -0.2737775146961212, -0.6450693011283875, 1.6278187036514282, -0.319044828414917, 0.8747720718383789, 0.5599024891853333, -0.8672301769256592, -0.3218299448490143, 0.14818456768989563, -1.533829927444458, -0.8290261626243591, -1.3661869764328003, 0.228690043091774, -0.6048594117164612, -0.7579305171966553, 2.0573346614837646, 0.7114750742912292, 0.32690948247909546, 0.41495808959007263, 0.4403054118156433, -0.8295515179634094, -0.6616209149360657, -2.742070436477661, -0.15658430755138397, 1.7219593524932861, -0.3414168655872345, 0.20579659938812256, -0.14966939389705658, -0.06946219503879547, 0.5703330636024475, -1.3021169900894165, -0.31721732020378113, -0.6858510375022888, 0.2525991201400757, -1.1682873964309692, -0.16069956123828888, -0.8082481622695923, -0.863965630531311], [0.45481061935424805, 0.2702830731868744, -0.25231897830963135, -2.704362392425537, 0.6222876906394958, 1.3695577383041382, -0.6912870407104492, 0.6779739856719971, -0.2725471258163452, -0.3612286448478699, 1.0863797664642334, -0.9161046147346497, -0.2271515280008316, -1.7116276025772095, -0.08266521245241165, -1.7059907913208008, 0.4892538785934448, -0.6088805198669434, -0.49276554584503174, 0.2781936824321747, 0.7949604988098145, -1.3882901668548584, 0.06295030564069748, -0.25712519884109497, -0.07874763756990433, 0.9509640336036682, -1.7820743322372437, -0.6257526874542236, -0.06511993706226349, -1.687731146812439, -1.5645604133605957, -0.3400859534740448, -0.5107274055480957, 1.073274850845337, -0.7604565024375916, 0.4584660828113556, -1.1032874584197998, -0.7173382639884949, 0.4376998841762543, 0.5906179547309875], [-0.9873830676078796, 0.5614659786224365, 0.1075761467218399, 1.3740020990371704, -2.1976711750030518, -1.3680412769317627, -0.8658890724182129, 0.4338853359222412, 1.1838738918304443, 0.2565545439720154, 0.18105077743530273, -0.25698643922805786, 0.1555352807044983, 0.7015472650527954, -0.09405805915594101, 0.3779243230819702, 0.4946422576904297, -1.879454493522644, -0.10823944956064224, 0.7201292514801025, 0.5404651165008545, -0.3312011659145355, 0.2541883587837219, -1.5200343132019043, 0.037584565579891205, 0.2265835404396057, 0.3726586103439331, -1.3773748874664307, -0.29657161235809326, 0.1000465601682663, -0.9911767244338989, -0.18519091606140137, 2.4070799350738525, -1.111480712890625, -1.371956706047058, 0.8611777424812317, -1.0880355834960938, 1.9344501495361328, 1.1682912111282349, -0.3388890326023102], [-0.17608611285686493, 0.5560617446899414, -0.5303480625152588, -1.1133602857589722, -0.7755382657051086, -0.2611696422100067, -0.25483736395835876, -1.7600014209747314, -0.9152507781982422, 0.8676446080207825, 0.039757758378982544, 0.09419946372509003, 1.679123044013977, -0.04620811343193054, -1.3751991987228394, 0.012143991887569427, 1.7688928842544556, 0.7513490319252014, 0.19847595691680908, -0.424252986907959, 0.0007118918583728373, 1.3907616138458252, -0.1688167154788971, 0.3696073293685913, -1.3562853336334229, 1.4305579662322998, -0.8225247859954834, -2.111112117767334, -0.36750802397727966, -0.38085243105888367, 0.3536737859249115, -0.12383758276700974, 0.4250582754611969, 1.285994052886963, -0.9203583002090454, -0.9635184407234192, -0.9134917259216309, -1.0132060050964355, 1.5168507099151611, -1.1696436405181885], [0.2777516543865204, -1.4322484731674194, -0.55910325050354, 0.0010511853033676744, 0.4893435835838318, -0.4932832419872284, -2.959752321243286, 0.09208710491657257, -0.4019100069999695, -1.495597243309021, -0.18005646765232086, -0.427255779504776, -0.22860756516456604, 0.5683399438858032, -1.0149308443069458, -1.7747730016708374, -0.48895880579948425, 0.7049499154090881, -0.09264987707138062, 0.504545271396637, 0.6191263794898987, -0.2614615261554718, 0.11852546781301498, 1.7485038042068481, 0.2330392301082611, -1.5859724283218384, 0.4563042223453522, -0.8964616060256958, -0.5471675992012024, -0.29959025979042053, -1.2903331518173218, -2.056602954864502, 0.3541233241558075, 0.8232714533805847, -1.4201298952102661, 0.7580833435058594, -0.8244582414627075, -0.8683607578277588, 0.015432588756084442, 2.3792543411254883], [-0.28691619634628296, -0.950371503829956, -0.9607475996017456, 0.06099586933851242, 0.9945323467254639, -2.7910754680633545, 0.8252919912338257, 0.7818968296051025, -0.02991056628525257, 1.5283550024032593, 0.5488404631614685, -1.0136033296585083, -0.3157240152359009, -0.9066669940948486, -0.9963409900665283, -0.6131567358970642, 0.167959064245224, 1.368645429611206, -1.724860668182373, -0.13693705201148987, 1.0122191905975342, 0.9716246128082275, 0.9917385578155518, 1.2512037754058838, 1.0513904094696045, -0.11216242611408234, -0.709984302520752, 0.43911606073379517, 0.8132602572441101, -1.4201159477233887, 0.7430119514465332, 0.3165616989135742, -0.1974199116230011, -0.43428835272789, 0.3970688283443451, 0.12197086960077286, -1.7002078294754028, -0.26203975081443787, -0.42806342244148254, 2.1584014892578125], [0.4700610637664795, 1.077859878540039, -0.5195188522338867, 1.2011797428131104, 0.29537421464920044, 0.5113298892974854, -0.5967219471931458, -0.08557320386171341, -0.8115701675415039, 0.05588105320930481, 0.8134140372276306, -0.09491093456745148, -1.2035225629806519, 0.6921157836914062, 0.46283653378486633, -0.5377913117408752, 0.7766953110694885, 0.16225068271160126, -1.0109951496124268, -0.4011963903903961, -0.8504109382629395, -0.8280924558639526, -1.4445691108703613, -0.5493254661560059, 0.8237364888191223, 0.2254093438386917, -0.9369168281555176, 0.20484177768230438, -1.3601586818695068, 0.21019615232944489, -0.6838464140892029, -0.47165483236312866, -1.7588739395141602, 0.45836755633354187, 0.15902186930179596, -1.6849857568740845, 0.5839415788650513, -1.937485694885254, 1.5471307039260864, -1.2229998111724854], [1.8288205862045288, 0.998274564743042, 0.3825731575489044, 0.5153747200965881, -2.207958459854126, 0.9204092621803284, 0.40903133153915405, 0.9520905017852783, -0.6463716626167297, 0.5311643481254578, 0.5531882643699646, 0.8146407008171082, -1.560301423072815, -2.670567035675049, -2.734875440597534, 0.5181509852409363, -2.3634231090545654, 2.9983608722686768, 1.441612958908081, 1.480086088180542, -0.8592644333839417, 0.4730921685695648, -0.6748006343841553, 0.019758516922593117, 0.27556368708610535, 0.3118272125720978, -0.5397859215736389, 1.1625850200653076, -1.1224143505096436, -2.9712650775909424, 0.23521798849105835, -0.8060618042945862, 0.2501615881919861, 1.4595779180526733, -1.0339546203613281, 0.12824159860610962, -1.0822319984436035, 1.5796326398849487, -0.43772825598716736, -0.996570348739624], [1.2144345045089722, 0.20662565529346466, -0.2623981237411499, -1.1948930025100708, 1.5039011240005493, -0.29786309599876404, 0.6553305387496948, 0.2799021899700165, -0.4384450316429138, -0.6497851014137268, -0.5012401938438416, 0.8178213834762573, 1.3169348239898682, 0.3697347640991211, 0.8402465581893921, -0.7226009368896484, -0.8992966413497925, -0.7535354495048523, 0.7041863799095154, -1.04374361038208, -1.4249122142791748, 0.9142698645591736, 1.3179491758346558, -0.00482054241001606, 1.2040809392929077, -1.0363491773605347, -0.3864589035511017, -1.0651236772537231, 1.4148118495941162, -0.09310249984264374, -0.0733954980969429, -0.8214921951293945, 1.019324779510498, 0.3083750605583191, 1.180072546005249, 0.6798543930053711, 0.5784313678741455, -0.513347864151001, 2.082695722579956, 1.1611534357070923], [0.530544102191925, 0.9712977409362793, -0.9884591102600098, 0.28813591599464417, -1.1659774780273438, -1.7615444660186768, -0.12548525631427765, 0.48854970932006836, -0.6167146563529968, -1.1501178741455078, -0.7256343960762024, 1.3618853092193604, 1.4775294065475464, -1.4755202531814575, -1.007678747177124, 0.8105040192604065, -0.11233928054571152, 0.49273574352264404, 0.11669738590717316, -0.484752357006073, 0.06738876551389694, -1.9402313232421875, -0.5701116323471069, 0.08499172329902649, -1.6471173763275146, 0.8466355204582214, -0.9097309112548828, 0.10028663277626038, -1.376432180404663, 1.296525239944458, -0.9272470474243164, 0.2685396671295166, 0.9050567746162415, -1.2861238718032837, -2.34177827835083, -0.26791125535964966, 0.13940179347991943, 0.5324254631996155, -0.4232119023799896, 0.08879196643829346], [0.5696607232093811, -0.4329921007156372, 1.0883492231369019, 0.17125913500785828, 0.15441790223121643, -1.3642005920410156, 0.9269551038742065, -1.1572484970092773, -0.3120392858982086, -0.3375217020511627, -1.2983897924423218, -0.5540663003921509, -0.7894911766052246, -2.285412311553955, -0.6193342804908752, -1.2120718955993652, 0.5793513655662537, 0.8985846638679504, -1.075080394744873, 0.3123353123664856, 1.0264180898666382, -0.7393922209739685, -0.383909672498703, -0.737348198890686, -0.3704555332660675, 0.732261598110199, 0.09587069600820541, 1.2738028764724731, 1.1665719747543335, 0.2688849866390228, -1.5979288816452026, -1.7439594268798828, 0.15256090462207794, -0.09211567044258118, -0.05316409841179848, -1.8150149583816528, -1.9004617929458618, -0.02902369759976864, 0.8075464963912964, -0.7923418283462524], [-0.7787591218948364, -1.9318379163742065, -0.640171229839325, -0.2388007491827011, 1.579907774925232, -1.0758405923843384, -1.073511004447937, -0.3747119903564453, -1.7709038257598877, -0.7890306115150452, 0.13427120447158813, -0.381117045879364, 0.01889733597636223, -2.4078948497772217, -2.597916841506958, -0.3492494821548462, -1.0042681694030762, -1.2486878633499146, 0.07257373631000519, -0.5189348459243774, 0.22662818431854248, -0.42179378867149353, -0.4705255329608917, -0.03285480663180351, -0.8796321153640747, -0.5492709875106812, 0.3497108221054077, -0.3352891504764557, -1.0319985151290894, -0.24699881672859192, -1.3132789134979248, -1.9217578172683716, 0.5450583696365356, 0.25050005316734314, -1.0196692943572998, 0.7298327088356018, 0.6278659701347351, 0.5075515508651733, -0.06074623391032219, -0.7323601245880127], [-0.6556490659713745, 1.5023146867752075, -0.5443021655082703, -2.193476438522339, -0.4187685549259186, 0.49635517597198486, -0.48066946864128113, 1.0796197652816772, 1.341435432434082, -0.7268271446228027, 2.021054744720459, 0.9808123111724854, 0.7783291935920715, -0.04596211016178131, -1.0538806915283203, 0.7037971615791321, 1.3377455472946167, -1.1703016757965088, 0.8301235437393188, -0.4580831825733185, -0.18796056509017944, 1.4070979356765747, -0.20164841413497925, -0.22542010247707367, -2.076873779296875, 0.8572480082511902, -0.0842595174908638, 0.6288527846336365, -2.321626663208008, -1.3621495962142944, -0.15406572818756104, 0.27532094717025757, 1.8807088136672974, 1.6300114393234253, 0.4514947235584259, 0.7692745327949524, 0.40199270844459534, 1.5918775796890259, -0.5089594721794128, -0.9441035389900208], [0.038625359535217285, -0.06344295293092728, 1.0740864276885986, 0.27654892206192017, -0.6327984929084778, 0.9697007536888123, -2.81929612159729, 2.333815097808838, 0.9864479303359985, -0.05503488704562187, 0.47932714223861694, -0.1497184783220291, -0.4626813232898712, -0.4542282819747925, -1.4987859725952148, 0.764905571937561, -0.9445632100105286, 1.0161899328231812, -0.39539796113967896, -1.0316238403320312, 0.5628805160522461, -0.09674859791994095, 1.2175159454345703, -1.7938756942749023, 0.5422132015228271, 0.5183690190315247, -0.9670534133911133, -0.5458592772483826, -0.8436306715011597, -1.414131999015808, -0.4161066710948944, -0.18199262022972107, 0.09513786435127258, -0.08576356619596481, -0.00511559471487999, 0.750571072101593, -0.07558419555425644, -0.5025731325149536, -1.008678674697876, 0.17458336055278778], [0.6752665638923645, -0.1748087853193283, 2.353609085083008, -0.9721885919570923, -0.17841298878192902, 0.15166844427585602, 0.28986656665802, -1.051922082901001, -0.1607801914215088, -0.09137773513793945, -0.40622326731681824, -1.7226169109344482, -1.3667794466018677, 0.6584638953208923, 0.43480566143989563, -0.9680196642875671, -0.017286531627178192, 0.7170146703720093, -0.4422794282436371, 0.20607274770736694, -0.2533715069293976, -0.5703971982002258, 0.3001134395599365, 1.2044330835342407, 0.895012378692627, 1.8543099164962769, -2.0924534797668457, -0.3893393278121948, 0.9429395794868469, -1.3655928373336792, -0.90821373462677, -1.1069233417510986, -1.6302961111068726, 0.8133360147476196, -0.31601428985595703, 0.5861946940422058, -1.5522184371948242, 0.8867940306663513, -0.09057524800300598, 0.3737071454524994], [0.6952812075614929, 0.689523458480835, -0.5200980305671692, 0.5522052645683289, -0.10923188924789429, 0.5437833070755005, -1.1172616481781006, -0.08448556065559387, 0.24502691626548767, 1.3012114763259888, 0.10797327756881714, 0.8427363634109497, -0.5202575325965881, -0.4845805764198303, 0.8533850908279419, -2.058872699737549, -1.324395775794983, -0.15248434245586395, -0.8415268659591675, -0.10932832211256027, -0.938011109828949, 0.7571538090705872, 0.39961153268814087, -0.45060351490974426, 0.3137213885784149, -0.6049531698226929, 0.05673215165734291, -1.4318808317184448, 0.1681395173072815, 0.013814643025398254, -2.2963666915893555, 0.9443143606185913, 0.19274859130382538, 1.6766986846923828, -0.7907719016075134, -0.23778408765792847, 0.11303402483463287, -0.13799576461315155, -1.0742419958114624, 0.2591608464717865], [1.3967140913009644, -0.009286506101489067, 2.774963140487671, -0.20841699838638306, 0.7640284299850464, -0.7368210554122925, 0.15115146338939667, 0.44000864028930664, 1.4743245840072632, 0.5143173336982727, -0.7360862493515015, -0.1933038830757141, -0.021478289738297462, 0.7012887001037598, 0.9150773882865906, 0.5030048489570618, 0.21073947846889496, -0.12211991101503372, 0.9883958697319031, -1.9326857328414917, 0.6576613783836365, 0.6107178330421448, -0.7906843423843384, -1.5881845951080322, 0.9629708528518677, -1.505260944366455, 1.9058985710144043, 0.8740925192832947, -0.9566776156425476, -0.13663458824157715, 0.8035975098609924, 1.7624112367630005, 0.0907491147518158, 1.5141360759735107, -0.1276278793811798, 0.3792741894721985, 0.5107842683792114, -0.9567248225212097, 0.0845143124461174, 0.5348297953605652], [-0.50129634141922, -0.8368451595306396, 0.3364774286746979, -0.4106076657772064, -0.016606368124485016, 1.9832282066345215, 0.7839483618736267, 0.1598283052444458, 0.9696909189224243, -0.10545854270458221, -0.6177355051040649, 0.1349503993988037, -0.4637143909931183, -0.5321409702301025, -0.8960391283035278, -2.6228926181793213, -0.3106382191181183, 0.2976146638393402, -0.9142261743545532, 0.17721840739250183, -2.0873069763183594, -0.29479148983955383, 0.7162562012672424, -0.9130960702896118, -0.9602458477020264, -0.5164109468460083, 0.45589613914489746, 2.607961654663086, -0.9833786487579346, 0.8011689782142639, 1.1553900241851807, 0.22570309042930603, -0.8974213600158691, -0.14503881335258484, 1.4127163887023926, -0.4408990740776062, -0.7703948616981506, -0.5182398557662964, -1.9156039953231812, -0.7462866902351379], [0.08891426771879196, -0.4151887893676758, -0.5187631845474243, -0.7276127934455872, 1.4160313606262207, -0.45474040508270264, 2.5600311756134033, 0.3815581798553467, 1.531185269355774, -1.3871854543685913, -0.23794463276863098, -0.45909568667411804, 1.0930349826812744, -1.1562589406967163, 1.3735142946243286, -0.7950565814971924, -1.1376487016677856, -0.10026779025793076, -0.4261223375797272, 1.7703213691711426, 2.34792423248291, -0.9931023716926575, -0.6468453407287598, -0.09752605110406876, 0.3030938506126404, 2.191725730895996, -1.6414783000946045, 1.5266132354736328, -0.4311378598213196, -0.4784512519836426, -0.16843348741531372, 0.46417108178138733, -0.06098313257098198, -0.569789469242096, 0.16531842947006226, 0.8182268738746643, -0.11579519510269165, -0.27599382400512695, -0.18902023136615753, 1.346264362335205], [-1.754032015800476, 2.0574803352355957, 0.09327509254217148, -0.3840176463127136, -1.1719732284545898, -0.08639440685510635, 1.8466582298278809, -0.0006621880456805229, 0.2952631413936615, 0.7731221318244934, -0.12397437542676926, -0.9932382702827454, -0.5412875413894653, -0.0853765532374382, 0.04880445450544357, -0.5722368955612183, 0.11601477116346359, -0.22658029198646545, 0.16511139273643494, -0.24864627420902252, 0.6428275108337402, -1.6873911619186401, 0.5362619757652283, -1.3051077127456665, 0.040456075221300125, 1.6029192209243774, 0.9260669946670532, 0.6683202981948853, -1.7678344249725342, 1.2394866943359375, 0.522749662399292, 1.193197250366211, 0.24014630913734436, -0.3045041859149933, -0.32614827156066895, -1.4360259771347046, -0.9169000387191772, 0.027814198285341263, 0.5688656568527222, 0.14077015221118927], [1.2885996103286743, -0.5791088938713074, -0.06261583417654037, 0.19957590103149414, -0.027960801497101784, 1.8022524118423462, -0.20723800361156464, 0.020758187398314476, -1.1132676601409912, -0.8269111514091492, 0.6502810716629028, 0.22107461094856262, -1.3163763284683228, -0.9858152270317078, 0.6882772445678711, 0.02981574833393097, 0.7643419504165649, -1.684783697128296, -0.6483622193336487, -0.20514445006847382, -0.6434341073036194, -0.1814345419406891, 1.5857259035110474, 2.0596792697906494, 0.05470145493745804, 1.0638989210128784, -1.4630708694458008, 1.265899419784546, 1.1994049549102783, -0.6307615637779236, -0.16350248456001282, -0.43651649355888367, -0.7038185000419617, 1.0837973356246948, -0.6810956001281738, 0.03574671223759651, -0.3182760179042816, -0.6446578502655029, 1.1447763442993164, 0.6416982412338257], [0.08304713666439056, 0.04561953991651535, 0.27592983841896057, 0.4357522130012512, 1.1209238767623901, -1.367207646369934, 0.9257816076278687, 0.22881555557250977, -1.7192193269729614, -0.9442898035049438, -0.9972198009490967, -2.004441976547241, -0.24423180520534515, -0.5653845071792603, -1.4205244779586792, -1.1179858446121216, 0.15786266326904297, -0.12475988268852234, 0.5753801465034485, 0.0034352897200733423, -0.25950145721435547, 1.3914055824279785, -2.4764063358306885, -0.2365078330039978, 2.32966947555542, -0.6517357230186462, 1.2821152210235596, 1.3601454496383667, -0.40188735723495483, -0.5526894927024841, -0.5668774843215942, 0.5339463353157043, -0.9334113597869873, -0.834974467754364, -1.070398211479187, -0.19301888346672058, -0.736013650894165, 0.23565317690372467, 1.0482137203216553, -0.663368284702301], [-0.49947431683540344, -1.5029240846633911, -0.43432334065437317, 0.8821106553077698, 0.3452516794204712, 0.7955151796340942, -0.9906445741653442, -0.10883013159036636, 0.9734048843383789, -0.19497381150722504, -1.10548734664917, 0.6824356913566589, 1.291764259338379, 1.2974432706832886, -1.0038708448410034, 0.24055682122707367, 0.18943007290363312, 0.706848680973053, 2.6441521644592285, -0.362297385931015, 1.580876350402832, 1.0791878700256348, 0.5572581887245178, 0.6893406510353088, -0.5056037306785583, -1.1927235126495361, -1.6573232412338257, -1.2206720113754272, -0.6479188799858093, 0.4746713936328888, 0.7000283002853394, 1.2553143501281738, -0.64137202501297, 0.48792505264282227, 0.4902805685997009, -0.3322702944278717, 0.5493173003196716, -0.8673111796379089, 2.8711025714874268, 2.9824867248535156], [1.038517951965332, 0.8812792301177979, -0.5008030533790588, 0.18061165511608124, -0.5015285015106201, 0.42860063910484314, 0.18763698637485504, -0.8586075901985168, 1.2820154428482056, -0.7919855117797852, -1.3715215921401978, -0.2071249634027481, 1.1841692924499512, 1.7276464700698853, 0.8726043105125427, -0.013412387110292912, 0.1508447825908661, -1.2028367519378662, -0.6629007458686829, 0.47776591777801514, -1.4574573040008545, -0.34590521454811096, 1.448049783706665, -1.4149742126464844, 0.6970107555389404, 0.48367640376091003, 1.07928466796875, 0.7456701993942261, 0.1050659641623497, 1.3799463510513306, 0.6628074049949646, -0.3525715470314026, 1.4913280010223389, 0.621749997138977, 1.221272349357605, 0.12743254005908966, 0.3617703914642334, 0.5885301828384399, -0.7924534678459167, 0.1542212963104248], [1.7063020467758179, 1.7538347244262695, 0.7970122694969177, 0.32982176542282104, -0.12886375188827515, -0.3648851811885834, -0.75770503282547, -0.4830951988697052, -1.2186506986618042, -1.0751643180847168, -0.9630376696586609, 0.4263513684272766, -0.42063289880752563, -0.4167774021625519, 2.274120807647705, -0.02247575670480728, 1.8970017433166504, -0.015716155990958214, 1.611003041267395, 1.3925172090530396, 0.5572433471679688, 1.6138050556182861, 0.23945371806621552, 1.654295802116394, -0.5861178636550903, -0.46834617853164673, -1.2070327997207642, 0.8239015340805054, 1.115420937538147, -0.042231202125549316, 0.08781600743532181, -0.327078253030777, -0.9318299293518066, -0.5514596700668335, -0.8298100233078003, 1.3968489170074463, -0.8614238500595093, -0.9835535883903503, -0.44660672545433044, -0.22815771400928497], [0.5181713104248047, -0.7519906163215637, 0.9508116841316223, -0.0015505359042435884, -0.41131776571273804, 0.9073682427406311, -0.2730284631252289, 0.09544425457715988, -0.7745803594589233, -0.4557700455188751, -0.3958922326564789, 0.7462933659553528, 0.6655317544937134, 0.02202785015106201, -0.40638551115989685, 0.6838915944099426, -0.4989606738090515, -2.1425414085388184, 0.267943412065506, -0.7377713322639465, 0.25789257884025574, -0.5473366379737854, 1.2498513460159302, 0.4701513350009918, -0.2707393765449524, 0.5845199823379517, -2.9294235706329346, 0.6047166585922241, -0.17579051852226257, 0.0888088122010231, 0.15909822285175323, -2.3379645347595215, -0.5279055833816528, -1.4391473531723022, 0.331765353679657, -1.1435426473617554, -0.5433085560798645, 0.43502312898635864, -1.5013662576675415, 0.9313440918922424], [-0.5105410814285278, -0.6487268209457397, -0.3289788067340851, -1.3987287282943726, 0.7962247729301453, 0.13435961306095123, -0.36063823103904724, -0.6137347221374512, 0.5405995845794678, -1.5907057523727417, 0.32703539729118347, 0.3122573792934418, -1.4375814199447632, 1.4933531284332275, 1.7467048168182373, -0.3132481873035431, 0.3450143337249756, 1.5628209114074707, 1.332349181175232, 1.4935091733932495, 0.5407619476318359, 0.4503287971019745, -1.3172452449798584, -0.0037879080045968294, -0.6351267099380493, -0.539351224899292, -0.6330069303512573, 0.15496838092803955, -0.09712190181016922, 0.9540621042251587, -0.7231836915016174, -0.26420995593070984, 1.2834852933883667, -1.6456010341644287, -2.037497043609619, 1.8554385900497437, 1.850733995437622, 0.8831735849380493, 2.2869620323181152, -0.6942160725593567], [1.1574018001556396, -0.6696470975875854, 1.4758416414260864, 1.2398247718811035, -0.012646850198507309, -1.4247199296951294, -0.5854135155677795, 0.4350602924823761, 0.6642345190048218, 0.5504884123802185, -0.5004978775978088, 0.571118175983429, 0.6912849545478821, -0.07958590239286423, -1.304192066192627, -0.10350063443183899, -0.32935574650764465, 0.35728421807289124, 1.0777835845947266, 0.12393521517515182, 0.3901738226413727, -0.3859086334705353, -2.0563805103302, 0.5978716015815735, 0.49393123388290405, -0.8598625659942627, -0.5917578339576721, 0.29500606656074524, -0.58804851770401, 0.25206291675567627, 0.7775940299034119, 0.8266938924789429, 0.11466043442487717, 1.9144439697265625, -1.7901033163070679, 0.8621123433113098, 1.1573673486709595, 0.13747327029705048, 0.5719529390335083, -0.6534742712974548], [0.22460110485553741, -0.9869459867477417, -1.1948295831680298, -0.13926082849502563, -0.030410418286919594, 0.1532798409461975, -0.2143123298883438, -0.66745525598526, 1.3797073364257812, 0.3548031747341156, 1.1943460702896118, 0.4489786624908447, -1.2646254301071167, 1.954856276512146, -0.33526232838630676, 0.12006015330553055, 1.4576033353805542, -2.203486204147339, -1.287842869758606, -0.10359936207532883, 1.5950051546096802, -1.267451286315918, 0.9339051246643066, -1.082812786102295, 1.6809526681900024, -0.49278077483177185, -0.36198264360427856, -1.3179421424865723, 0.27186280488967896, 0.4109951853752136, 0.2919866442680359, -0.8963986039161682, -0.2947309911251068, -0.5312963724136353, 1.4173229932785034, 0.837882936000824, 0.15565848350524902, -0.151556134223938, -1.459203839302063, -0.6654043197631836], [0.6857332587242126, -2.274278163909912, 0.06758692860603333, -0.04204070568084717, -0.6810629367828369, -0.03568049147725105, 0.17728273570537567, 1.2237745523452759, 0.7871622443199158, 1.5787343978881836, 0.9688153862953186, -0.6161239147186279, 0.6247457265853882, 0.5577312707901001, 0.5271569490432739, -0.8563572764396667, -0.06314210593700409, 1.2431071996688843, -0.965391993522644, 0.26610949635505676, 0.3390316665172577, -0.6250584125518799, -0.3134172260761261, 1.5168722867965698, -0.09427063912153244, 0.7698051929473877, -0.45900118350982666, 0.8319160342216492, -0.5436392426490784, -0.6179584860801697, -1.0969189405441284, -0.5528904795646667, -0.06999468058347702, -1.4763861894607544, 0.5502636432647705, -1.0865486860275269, 1.4330954551696777, 1.4158886671066284, -0.3289223909378052, 1.309175729751587], [-1.4615100622177124, 0.46414947509765625, 0.9276500344276428, 0.843791663646698, -0.6474283337593079, -0.3637043535709381, -1.6600061655044556, -0.38355210423469543, -0.11105155199766159, -0.9116191267967224, 1.3902333974838257, -0.7714371085166931, 0.564071774482727, 0.31777897477149963, -0.6769794225692749, 0.11309676617383957, 1.024734377861023, 0.741003692150116, -0.16389600932598114, -0.37572112679481506, 0.601456344127655, -0.03592377528548241, 0.06880711764097214, -0.7010525465011597, -0.004810337442904711, -0.15969008207321167, -1.0544975996017456, 0.6101467609405518, 1.2451677322387695, -0.06328694522380829, 0.012559758499264717, 0.5363026261329651, -0.30207526683807373, -1.339689016342163, -0.30540022253990173, 0.16918638348579407, 1.2677874565124512, 1.5709433555603027, -0.008915554732084274, 0.3295857608318329], [1.384276270866394, 0.09356982260942459, -0.8005232214927673, -0.08050121366977692, 0.6439758539199829, -0.1753360629081726, -1.2257424592971802, -0.6172826290130615, 1.1575919389724731, 1.5485981702804565, 1.6188277006149292, -0.5094849467277527, 0.4641272723674774, 0.7934365272521973, -0.319922536611557, -1.6978206634521484, 1.0805374383926392, 1.2901690006256104, 0.5536346435546875, 1.23139488697052, 0.3081666827201843, -1.1167123317718506, 0.32214969396591187, 0.03722621500492096, 0.761723518371582, -1.0583875179290771, 0.9535436034202576, 1.0548657178878784, -0.40682926774024963, -0.38660451769828796, -0.9745501279830933, 1.9486770629882812, 0.9676282405853271, -1.6242377758026123, -0.7381042242050171, -1.2299364805221558, 1.4803452491760254, 0.9123517870903015, -1.3963940143585205, 0.38049352169036865], [-0.1480119377374649, 0.3500898480415344, -0.35238152742385864, 1.4849504232406616, 0.9642974138259888, -1.1494394540786743, -1.013749361038208, 0.5290305018424988, 1.1172375679016113, -0.9820351004600525, -0.32035449147224426, 0.2983610928058624, -0.15120305120944977, 1.9055981636047363, -1.5621029138565063, 1.2118680477142334, -0.4607570469379425, 0.6618191003799438, 0.3035041391849518, 0.40880292654037476, -1.3098490238189697, 0.21182502806186676, -0.7091432809829712, -0.6942111253738403, 0.1733117401599884, -0.21316055953502655, 0.3114287853240967, 1.5833353996276855, 0.8167816400527954, -0.8016371130943298, -0.6384745240211487, 1.7269102334976196, -0.9329670667648315, 0.3710353672504425, 2.974571466445923, -0.6858358383178711, -0.3561376929283142, 1.0846925973892212, 0.7119312286376953, -0.7066296339035034], [-0.11666987091302872, 1.6543980836868286, -0.13381832838058472, -0.5094633102416992, 0.9008312821388245, 0.8240058422088623, 0.5322237014770508, 0.12414434552192688, 1.3398782014846802, -2.375941276550293, 0.4177643358707428, 0.5267214775085449, 1.450117826461792, 2.3224520683288574, -1.0673575401306152, -1.5494558811187744, 0.41149306297302246, 1.4813207387924194, 1.7921963930130005, 1.7414419651031494, 1.0602446794509888, 0.4730388820171356, 2.112126588821411, 2.0292551517486572, 0.2640388309955597, -0.18586869537830353, 0.009050542488694191, 0.7744850516319275, 0.03156713768839836, -1.427006721496582, -0.6581956148147583, -0.9480364322662354, -0.5036198496818542, -1.5767807960510254, -1.1012088060379028, -1.4303455352783203, 0.41439810395240784, -0.18898938596248627, -0.6643778681755066, 0.40249744057655334], [-0.4891734719276428, 0.10625196993350983, 1.462786078453064, -1.255561113357544, 0.3183463513851166, 0.5231963396072388, -1.4508908987045288, 0.09530143439769745, 0.32849061489105225, -1.7723256349563599, 2.3411269187927246, -0.2885909974575043, -0.47269532084465027, -0.8505148887634277, -2.5366971492767334, 0.5908929109573364, -0.3258203864097595, -1.296199083328247, 0.9150473475456238, 0.41974249482154846, -0.21013931930065155, 0.05078067630529404, -0.28035497665405273, -0.8425974249839783, 1.0510458946228027, 2.6233553886413574, 1.6069658994674683, -0.9160701632499695, -1.2378077507019043, -0.4354936182498932, -0.16975094377994537, 0.29663538932800293, -0.33755993843078613, 0.49473321437835693, 0.043773941695690155, 1.290045142173767, -0.1401161104440689, 0.14253954589366913, -1.077411413192749, 0.3697781562805176], [0.14750424027442932, -1.3650683164596558, 0.5227853655815125, 0.37283647060394287, -1.2666726112365723, 1.9825620651245117, 0.7898643612861633, -0.44510188698768616, -0.47496089339256287, -0.39948904514312744, -1.0099486112594604, 1.153112769126892, 0.09016536921262741, 1.5908663272857666, 2.1162898540496826, 0.39068180322647095, 1.1616648435592651, 0.27607855200767517, 0.4956836700439453, -0.7752244472503662, 1.1348049640655518, 1.0807931423187256, 0.39342883229255676, -0.6689210534095764, -2.4717867374420166, 1.5201581716537476, -0.1109776720404625, -0.4791717529296875, 0.26632195711135864, -1.195948600769043, -0.30025097727775574, 0.13265033066272736, -0.6564551591873169, 0.021428218111395836, 0.98917555809021, 0.0774444192647934, 0.5094159841537476, 0.0016969740390777588, -0.9844586849212646, 0.5908551812171936], [-1.26997709274292, 0.8288754224777222, -0.4368368089199066, 0.8019987940788269, -1.9561320543289185, 0.291411429643631, 1.2736896276474, 3.0294272899627686, -0.8021997213363647, 2.272854804992676, -2.217072010040283, 0.49712395668029785, -0.6435525417327881, -1.6127829551696777, -0.1451674848794937, -1.3738566637039185, 0.991087794303894, 0.6041918396949768, 1.6941605806350708, -0.022306887432932854, 0.3819054663181305, 1.005476474761963, 1.8321045637130737, -1.494886040687561, -0.7138226628303528, 0.828051745891571, -0.3898300230503082, 0.7236557006835938, 0.8142156004905701, 0.07322584837675095, -0.8917774558067322, 0.6592589616775513, -1.9211692810058594, 1.2496494054794312, 1.179487943649292, -2.760112762451172, 1.2761456966400146, -2.119762420654297, -0.7597390413284302, 0.7373948693275452], [-0.23209455609321594, 0.998907208442688, 1.3025262355804443, 0.9266389012336731, -0.7212857604026794, 0.10884121060371399, 1.3873865604400635, 0.3567901849746704, -1.0809305906295776, 0.4039127826690674, 0.021408507600426674, 0.5388200283050537, 0.4847598969936371, -0.13681937754154205, -0.17437997460365295, 1.5994318723678589, 0.2598211467266083, -2.2057852745056152, -1.7028849124908447, 1.0351530313491821, 0.7970185279846191, -0.09882772713899612, -0.48446083068847656, 1.9964507818222046, -0.671725332736969, 0.365817129611969, -0.3297000229358673, -0.12459153681993484, -1.0918813943862915, -0.29642513394355774, 1.126556634902954, -0.6845508217811584, 1.2586463689804077, -1.7324519157409668, 0.17116855084896088, 0.7856453657150269, 0.1249297633767128, -1.0024558305740356, -1.5307668447494507, 1.2648338079452515], [-1.1877131462097168, 0.9023523926734924, 0.4138846695423126, -1.1340590715408325, 1.3391401767730713, -1.0781508684158325, 0.14351318776607513, 0.46312215924263, -0.47052881121635437, 0.6895595192909241, -0.6540961861610413, 0.6643867492675781, -2.1936452388763428, 0.11569739133119583, 0.8435363173484802, -0.6774402856826782, -0.7135112881660461, 0.17424820363521576, -0.7055224180221558, -2.3314340114593506, -1.232924222946167, -0.73572838306427, 0.710945188999176, 0.6936859488487244, 1.042737603187561, -0.3599127531051636, 0.23927843570709229, 0.20066983997821808, -0.22992005944252014, 1.7616140842437744, -0.4471619129180908, -0.11544457077980042, -1.3512605428695679, -0.8076685667037964, -0.19435463845729828, 0.21366186439990997, 0.391346275806427, 0.24091534316539764, -0.2677055895328522, -1.5726332664489746], [-0.44915229082107544, 1.3523513078689575, 0.37322378158569336, -0.616458535194397, -0.1774667352437973, -0.7274453639984131, -0.8946685194969177, 0.6076711416244507, 0.20522958040237427, 0.6477398872375488, 0.46401071548461914, -0.7220668196678162, -0.002182174241170287, 0.9628537893295288, -0.3753511905670166, 0.5115013122558594, -0.3851953148841858, 0.8070966005325317, 1.4577103853225708, 0.7984222173690796, 0.142813578248024, -0.9445615410804749, -0.31872305274009705, -0.535115122795105, -0.7349544167518616, -0.03507469221949577, 0.43011772632598877, 1.7534619569778442, -0.21825265884399414, 0.4633006155490875, -1.7969434261322021, -0.2830685079097748, 0.6641814112663269, -2.483013868331909, 0.5716568231582642, -0.17537282407283783, -0.5321816205978394, 0.738277018070221, -0.5067470669746399, 0.7616938948631287], [1.1073466539382935, 0.8261195421218872, 0.996668815612793, -0.2178812026977539, 1.2159026861190796, 1.1360992193222046, 1.3152341842651367, -0.5884549021720886, -0.4904201030731201, 0.8244487643241882, -0.5846028923988342, -0.35227206349372864, -0.5223461389541626, -1.7717783451080322, -1.2006691694259644, 0.5079373717308044, -0.7292625904083252, 0.5808238983154297, -0.11899823695421219, 1.266697883605957, -1.3128979206085205, -0.2823896110057831, -0.013395708985626698, 0.07756508141756058, -2.142427682876587, -1.1001665592193604, 0.2368537336587906, 0.46458718180656433, -0.48133015632629395, -0.7104268670082092, 0.9861854314804077, 0.051154181361198425, -0.7511732578277588, -0.22535091638565063, 0.990265429019928, 1.047663927078247, -0.332806795835495, -0.2789919972419739, -0.2050851434469223, 0.5079649686813354], [1.182608962059021, 1.0054726600646973, 3.090625286102295, -0.7606329321861267, 0.6397493481636047, -1.4721488952636719, 0.8902660012245178, 0.8031140565872192, -0.18584497272968292, -0.46272334456443787, 0.40083563327789307, -0.2719309329986572, -1.3540430068969727, 1.7785439491271973, -0.056866370141506195, -0.7573493719100952, 0.44503873586654663, 1.4427677392959595, 0.9188911318778992, -0.42759963870048523, 0.05232902988791466, -0.18053247034549713, 1.9612808227539062, 0.6650487184524536, 0.43108856678009033, 0.09275972098112106, -1.2048046588897705, 0.8216871023178101, 0.0872226133942604, -0.20201872289180756, 0.7105000615119934, -0.10563002526760101, 0.23644988238811493, -0.19920620322227478, 0.6261327862739563, 1.0883235931396484, -1.5641682147979736, 0.3571477234363556, -0.3279920816421509, -0.6631046533584595], [-0.03210929408669472, -0.86725914478302, -0.4881061911582947, 0.9199837446212769, -0.47982150316238403, -1.6893147230148315, -0.7506335377693176, -0.4047427177429199, 0.42902514338493347, -1.4984031915664673, -0.528638482093811, -0.006495001260191202, -0.7616409659385681, 0.48642632365226746, -0.18359455466270447, -0.23766210675239563, 0.26947516202926636, -0.4962819814682007, 0.2766978144645691, -0.7472970485687256, -0.7594730854034424, 0.32066819071769714, -0.9509151577949524, 2.209622621536255, 0.05099407583475113, -0.3708866238594055, -0.6548674702644348, 1.966104507446289, 1.3670017719268799, 2.198711633682251, -0.02527262642979622, -0.003017131006345153, -0.3390161991119385, -2.422891616821289, 0.6074999570846558, 0.8801016211509705, -1.069341778755188, 1.336606740951538, 0.4290331304073334, 1.327650785446167], [-1.9042408466339111, -0.14836075901985168, 0.679255485534668, -0.042921002954244614, -2.184782028198242, -0.23402293026447296, -1.3729946613311768, 0.019067933782935143, 0.9113526940345764, 1.7578121423721313, -0.11211469769477844, -1.4671432971954346, 1.0608313083648682, 0.24782730638980865, 0.8459515571594238, 0.44091731309890747, -0.10788403451442719, 0.2929925322532654, 1.4715710878372192, -0.6646401286125183, -0.19644445180892944, -0.05413428694009781, -1.208787441253662, -0.5570735335350037, 0.9363019466400146, -1.7689255475997925, 0.08014007657766342, -1.1310299634933472, 1.4119749069213867, -0.06800704449415207, 1.2684168815612793, 0.14017556607723236, 0.7504749894142151, 0.702979326248169, -0.052329882979393005, 1.2980358600616455, -1.5265519618988037, -1.7256942987442017, -0.34877821803092957, 0.27467069029808044], [-0.007311153691262007, 0.28907036781311035, 0.025444751605391502, -0.04423176869750023, 0.27234211564064026, 1.2827119827270508, -0.6082302331924438, 0.8232911229133606, 0.28252068161964417, 0.0804557278752327, -1.2375266551971436, -0.6608200669288635, 0.8759087324142456, -2.436319589614868, 0.07853594422340393, 0.3805500566959381, -2.3360183238983154, -1.044747233390808, 0.08340778201818466, -0.8204854130744934, -1.4096195697784424, -0.5856717228889465, 1.0832785367965698, -0.014084518887102604, 1.2367024421691895, -0.6683038473129272, 0.19419844448566437, -0.9269750714302063, -0.9807994365692139, -1.3115702867507935, 0.8587774038314819, 1.081776738166809, -1.232725977897644, 0.9922982454299927, 0.2770650386810303, 0.23651185631752014, -0.33823755383491516, -2.1315653324127197, 0.7114644050598145, -1.193537712097168], [-0.013445515185594559, -0.8291558623313904, 0.06510624289512634, -1.2225747108459473, 1.6131418943405151, 0.6977417469024658, -1.5247968435287476, 0.23033477365970612, 0.5297701954841614, 0.8068113923072815, 0.8848688006401062, -1.4503953456878662, 0.6954331398010254, 0.7451450228691101, 2.4463727474212646, -0.15929937362670898, 0.5611247420310974, 1.840194582939148, 0.5033682584762573, 0.16573107242584229, -0.09532646834850311, 0.3183383047580719, -0.6994820833206177, 2.2619192600250244, -0.8007922768592834, 1.7823189496994019, 0.10258007794618607, 1.2490599155426025, -0.8878181576728821, -0.994281530380249, -0.8374155163764954, 0.3762873113155365, -1.0916571617126465, -1.0382698774337769, 1.3806706666946411, 1.3375533819198608, -1.0803638696670532, 1.15068781375885, -0.35087186098098755, 0.6568649411201477], [-0.7659655213356018, 0.7871764302253723, -1.3643786907196045, -0.45110780000686646, 0.285839706659317, 0.17702335119247437, -0.014698970131576061, 1.6386109590530396, 1.109795093536377, 0.2605133056640625, 0.18175600469112396, -1.0844447612762451, 0.5930749773979187, -0.7919460535049438, 0.21541902422904968, 0.6480292081832886, -0.2276962846517563, -0.10020526498556137, 1.734971046447754, -0.8766082525253296, 1.3431313037872314, 0.45448824763298035, -0.3782346546649933, 0.2907617688179016, 0.28056660294532776, 1.19709050655365, -0.10793375968933105, 0.20991970598697662, -0.35788238048553467, 0.7066217064857483, -1.6654466390609741, -0.1650284081697464, -0.12531636655330658, -0.479962021112442, 0.016708938404917717, 1.6378357410430908, -0.9452492594718933, 0.5666496157646179, -1.381645917892456, -0.18410249054431915], [0.9443961381912231, 0.982836902141571, -0.9249843955039978, -0.516514003276825, -0.4489189684391022, -0.7077688574790955, 0.41202110052108765, 2.162670373916626, -0.44327548146247864, -0.5940483808517456, 0.4609256386756897, 1.1299028396606445, 1.3577715158462524, -1.4754992723464966, -0.3826889395713806, -0.32107844948768616, 0.5867278575897217, -1.17011559009552, -1.3932355642318726, -0.6662461161613464, -0.07738099992275238, -1.0527583360671997, -2.3846824169158936, -1.0983701944351196, -2.0781784057617188, 0.37348470091819763, 1.2806504964828491, -0.4318670332431793, 0.37222638726234436, -0.6821210980415344, -0.7019736766815186, -1.8572516441345215, 0.27786850929260254, -0.8683015704154968, 1.910459280014038, 0.932576596736908, -0.4032818377017975, -0.18358327448368073, 0.07283350825309753, -0.3600628077983856], [0.9366858005523682, -1.1015070676803589, 0.0003751635958906263, 0.529518187046051, 0.8369041085243225, 0.49300065636634827, -1.214093804359436, 2.5854291915893555, 0.6073986887931824, -0.10378776490688324, 1.4961864948272705, -0.5427718758583069, -0.46556776762008667, -0.5512027144432068, 0.9211107492446899, 1.7835062742233276, 0.7507457733154297, 0.7533829808235168, 0.1395491659641266, -0.42868009209632874, -0.1931786835193634, -1.2440298795700073, -0.13314826786518097, 0.3696754276752472, 3.1288888454437256, -0.6836845278739929, 1.2874886989593506, -0.16428790986537933, -0.3147885501384735, 0.8302323222160339, 1.0468379259109497, 0.7769300937652588, -0.4521922469139099, 1.1758217811584473, 1.0214332342147827, -0.9342454671859741, -0.09105395525693893, 0.6639015674591064, 2.0211129188537598, 1.342519760131836], [0.13705168664455414, 0.19221307337284088, 0.5547624826431274, 0.6581443548202515, 0.5826362371444702, 0.141343355178833, -0.7950811982154846, 1.0335822105407715, -0.19804194569587708, 1.3663055896759033, -0.5740005970001221, 0.6142882704734802, -0.3800284266471863, 0.6092487573623657, -0.46016812324523926, 0.19066400825977325, 1.7158712148666382, 1.0209728479385376, 0.30986231565475464, 2.6759419441223145, -1.0973659753799438, -0.15320883691310883, -1.203437089920044, 0.4526924192905426, 1.7293932437896729, -1.1732484102249146, -1.2686214447021484, -0.7376947999000549, 2.0094103813171387, -0.7454977035522461, 0.7765296697616577, 1.7431013584136963, -1.196200966835022, 1.1261723041534424, -0.016696566715836525, -1.432636022567749, 0.828161358833313, -0.5554818511009216, 1.0049046277999878, -0.012824957258999348], [0.1807689368724823, -0.7659717798233032, 0.7877479791641235, 0.5504748225212097, 0.17891669273376465, 0.23791532218456268, -0.7815680503845215, 0.6996649503707886, 1.8806123733520508, -1.0377111434936523, 1.2068403959274292, -1.0131350755691528, 0.9685481190681458, -0.24218805134296417, -2.516786575317383, 1.0387816429138184, 0.9037815928459167, -0.9310328960418701, 0.07628840953111649, 0.35633689165115356, 1.4927167892456055, 0.5487848520278931, 0.00944293662905693, 2.2299253940582275, 0.722766637802124, -0.757750928401947, -1.0162068605422974, -0.4356805980205536, -1.7279049158096313, 0.9874292612075806, 0.3970283269882202, 0.49900588393211365, -0.6876271367073059, -0.04318369925022125, -0.24313384294509888, 1.2641571760177612, -0.27607500553131104, -0.01351962611079216, 0.7426314353942871, 0.24814915657043457], [0.2743456959724426, 0.023885274305939674, 1.0682605504989624, -1.9435044527053833, -0.6228473782539368, -1.20722234249115, -0.30335190892219543, -3.03167462348938, -0.8336367607116699, -0.06098190322518349, 0.6031596064567566, 2.509556531906128, -0.5297191143035889, 0.2975756525993347, -0.5688321590423584, 0.961128830909729, -0.2587366998195648, 0.738309383392334, 0.8891557455062866, 0.5901389718055725, -2.4897654056549072, -2.1972551345825195, 1.053030252456665, 0.7736260294914246, -0.17491842806339264, 0.2557840049266815, -0.22893746197223663, -0.48160064220428467, 1.9587647914886475, -1.216368317604065, -0.8802452683448792, 1.2800343036651611, 0.16987696290016174, -1.1155582666397095, 1.8305338621139526, 0.8707744479179382, -1.1522125005722046, 1.5822474956512451, 0.8256847858428955, -0.526904821395874], [1.2081700563430786, -1.3559620380401611, 0.685298502445221, -0.07284785062074661, 0.8199102282524109, -0.13102641701698303, -2.4193997383117676, -1.3948661088943481, 0.3553202450275421, -1.5392050743103027, 0.03228987380862236, -2.0026328563690186, -1.0889440774917603, 0.08255793899297714, -2.1008152961730957, -0.7303441166877747, 0.15100747346878052, -1.4068903923034668, -0.47102925181388855, 2.1143815517425537, -1.639223575592041, -1.3449137210845947, 1.2262871265411377, -0.6658264398574829, -0.30467188358306885, 1.3834202289581299, -0.5043236613273621, -0.6431193351745605, 0.42054063081741333, -1.0165438652038574, -1.3101763725280762, 0.3516649603843689, 2.5833191871643066, 0.4677979648113251, 1.6083762645721436, 0.18773742020130157, 1.212112545967102, -0.03273908421397209, -0.4287848174571991, 0.8901078701019287], [-0.26998215913772583, -0.32207190990448, 1.297240972518921, 1.179574966430664, 0.06870065629482269, -0.26926669478416443, 2.546391010284424, -2.5746729373931885, -1.4945433139801025, -0.6722304224967957, -1.4725189208984375, 0.4473246932029724, 2.717055082321167, -0.05601399391889572, -1.3592987060546875, -0.11826906353235245, -0.09967467933893204, 0.40277600288391113, -0.6303017139434814, 1.0025173425674438, -0.29622361063957214, -0.7387745976448059, -0.09481876343488693, -1.180851936340332, -0.793859601020813, 0.5795571208000183, 0.6946377158164978, 0.8070852160453796, -0.9527637958526611, -1.3785057067871094, -0.8155333995819092, 1.3089829683303833, -0.42078697681427, 0.8701639175415039, 0.10468465834856033, 0.4534264802932739, -0.4157837927341461, 0.8331146240234375, 1.3299449682235718, -0.6885417699813843], [-0.4845549762248993, 0.21219155192375183, 1.2246346473693848, 0.6345183253288269, 1.6623324155807495, -0.9117385745048523, 0.7451801896095276, 1.9932223558425903, 0.11790131777524948, 0.5627956986427307, 0.4111310541629791, -0.013514173217117786, 0.13752488791942596, -0.7119420170783997, 0.4520626962184906, 1.4419108629226685, -0.35027873516082764, -0.0031978997867554426, 0.2616599202156067, -1.0607889890670776, -2.2495546340942383, -0.010744718834757805, 1.0439001321792603, -1.4848695993423462, -0.8094387650489807, 1.0831049680709839, 1.7966878414154053, 2.2051796913146973, -0.18639639019966125, -0.4180692136287689, 1.253591775894165, 1.139999508857727, -0.1222691684961319, -1.4625250101089478, -0.3771418035030365, 0.3671117424964905, -0.4497063159942627, -1.3771010637283325, -0.13719040155410767, 0.6243383288383484], [-1.0834628343582153, 0.8961546421051025, 0.5914850234985352, -1.150647759437561, -1.1815454959869385, -2.3209285736083984, 0.02802327089011669, -0.2061246633529663, -1.1660460233688354, -0.42543044686317444, 0.9067097902297974, -0.5932843685150146, -1.0144946575164795, 1.2559117078781128, -1.8634333610534668, 0.7550355195999146, -0.07598099112510681, -0.4790564477443695, -1.2396035194396973, 2.026090621948242, 0.0004023422079626471, 2.1578211784362793, 0.7291843891143799, -0.7793294787406921, -0.09901995956897736, -1.7042911052703857, 0.043907951563596725, -0.8222332000732422, 0.5683412551879883, -1.3697874546051025, -0.16037732362747192, -0.07750630378723145, -0.26969602704048157, -0.617576003074646, -0.4857487976551056, 0.5791007876396179, 0.313714861869812, -0.28649240732192993, 0.6967926621437073, 0.510823667049408], [1.0641928911209106, 0.9878349900245667, 1.1009348630905151, 0.6172094345092773, 0.14574934542179108, 0.786902666091919, -0.37762418389320374, 0.8494951128959656, -1.0949363708496094, -0.4584757387638092, -1.4980762004852295, -0.3371551036834717, -0.4209246039390564, 0.44415801763534546, 0.1424061506986618, -0.35549241304397583, 0.3551667630672455, -1.647985577583313, 1.6476249694824219, 1.60568368434906, -0.5209791660308838, -0.22259587049484253, -0.4944106936454773, -0.7240687608718872, 2.737281084060669, -0.7326761484146118, 1.0310418605804443, 0.5197526216506958, -0.8693504333496094, -2.195493459701538, 0.440397173166275, 1.916150450706482, 0.02493651956319809, 0.7391042113304138, 0.759513795375824, 0.5891627073287964, 0.6272132992744446, -0.8885985612869263, 0.3959730267524719, 0.5224551558494568], [0.5488635301589966, -1.190696120262146, -0.9695672392845154, 0.8472025990486145, -0.5533133745193481, 0.39927834272384644, -0.6793291568756104, -0.4327695667743683, -0.4069666862487793, -0.5308090448379517, -0.2110310047864914, -1.1825562715530396, 1.3413071632385254, 0.43660879135131836, -0.6887872219085693, 0.3527787923812866, 0.985243022441864, 1.1891868114471436, 0.29189345240592957, 0.27852991223335266, 2.309781074523926, 0.3258481025695801, 1.241919994354248, 0.2694718837738037, 0.8789376020431519, -0.7889289855957031, 1.1575558185577393, 0.28481605648994446, 0.413845032453537, 0.6599065065383911, -1.8837604522705078, 0.21129892766475677, 1.0729924440383911, 0.2941089868545532, -1.4846456050872803, 0.27613601088523865, -0.053214773535728455, 1.3215258121490479, -1.4004359245300293, 0.3128792345523834], [0.6539582014083862, -1.4535117149353027, -1.3612730503082275, -0.55522221326828, 0.9715278744697571, 1.4097265005111694, 0.6601097583770752, -0.7118968963623047, 0.9630743861198425, -0.3035697042942047, 0.3115565776824951, 0.7435973882675171, 0.7618915438652039, 0.03543613478541374, 1.0022728443145752, -0.9283661842346191, 1.3919655084609985, -0.7107223868370056, 0.6347723603248596, -0.16676832735538483, -0.8941746950149536, -1.5889195203781128, -0.13403616845607758, 0.4902525246143341, -0.7213388085365295, 0.10056153684854507, 0.9381645321846008, 1.1877638101577759, -1.134202480316162, 1.6103535890579224, 0.4374454617500305, -0.08889105916023254, 0.428178995847702, 0.6297916173934937, -0.016448402777314186, 0.9094136357307434, 0.8139854669570923, 0.5798831582069397, -0.08691928535699844, 1.0738141536712646], [0.6075400710105896, -0.8081822395324707, 0.9907221794128418, 0.7843062281608582, 1.262181282043457, 0.6669062972068787, -0.3095124363899231, -0.27621719241142273, -0.3637692630290985, -0.5672370791435242, 0.5544625520706177, 0.45477721095085144, 0.8880992531776428, -0.21893124282360077, 0.9719706177711487, -0.4004426896572113, 0.8484225273132324, 0.8150629997253418, -1.1753058433532715, -0.13190868496894836, 0.5168027877807617, -0.2792561650276184, -0.6406252980232239, 0.4903866946697235, -0.5718790292739868, 2.142333507537842, 1.1042064428329468, -0.7813224196434021, -0.7521296739578247, 0.21398493647575378, 0.7950950264930725, 1.624009370803833, -0.8650757074356079, -0.575480580329895, 0.1446719616651535, 0.79352205991745, 0.05114445835351944, 0.2494158297777176, 0.03908415511250496, -1.204773187637329], [-0.19712477922439575, 0.48727986216545105, -1.3049886226654053, 3.227203130722046, 0.7429316639900208, -0.7396788597106934, -0.7458135485649109, -0.9269596338272095, -0.5584617853164673, -0.446391761302948, 0.8075289130210876, 0.766998827457428, 0.4680885076522827, 0.02291201800107956, -1.062058687210083, 0.45419853925704956, 0.07863789796829224, -0.18622365593910217, 0.6764779090881348, -0.8832268118858337, 2.5995829105377197, 0.6190571784973145, 0.48998892307281494, 0.7357076406478882, -0.37273991107940674, -0.7503684759140015, -1.1820377111434937, 0.9384185075759888, 0.14172756671905518, -0.01545734889805317, -1.0906327962875366, 0.9117516875267029, 0.8669506311416626, 0.40109768509864807, -0.3773055672645569, -0.6110855340957642, 0.4355930984020233, 0.6281524300575256, -0.4064881205558777, -1.34865403175354], [0.18341636657714844, -0.7602739930152893, 1.2796814441680908, -0.22654041647911072, 0.30051958560943604, 1.386995792388916, -0.07411112636327744, 0.5812356472015381, 0.6129153370857239, -0.39906033873558044, 0.8560254573822021, -1.0820029973983765, -0.5398434996604919, -0.5134536623954773, -1.5861104726791382, 1.5370204448699951, -1.761797308921814, -1.5170323848724365, -0.5018578767776489, 0.22874003648757935, 1.1234785318374634, 1.5098375082015991, -0.2794685363769531, 0.4073033630847931, -1.2052571773529053, 0.7530972361564636, -0.27996429800987244, -0.14281736314296722, 1.615537166595459, 1.2111632823944092, 0.02802463434636593, 0.456562340259552, -0.399093896150589, 0.45936790108680725, -0.07091294974088669, 0.23237094283103943, 2.7548930644989014, 1.9443626403808594, 0.6572408676147461, 1.4381014108657837], [-0.21774710714817047, 1.1208101511001587, -1.6127394437789917, 0.5923665761947632, -0.6983866095542908, 0.07061956077814102, -0.5056829452514648, -0.5222582221031189, 0.31416380405426025, 1.0099188089370728, -1.3342195749282837, 1.2048640251159668, -1.0389221906661987, -2.294861316680908, -0.9124125838279724, -0.8187703490257263, 0.9321784973144531, 0.20831097662448883, 0.44495463371276855, -0.7794371247291565, -0.12596800923347473, -0.3937639594078064, 0.5713986158370972, -0.4099990725517273, 0.40662094950675964, -0.7502681016921997, -0.4715968072414398, -0.10342234373092651, 0.2939426004886627, -0.5474522709846497, 0.9915997982025146, -0.6141344308853149, -0.08592715859413147, 0.0769423097372055, -0.8172793388366699, 0.15476304292678833, -0.6279581189155579, 2.183180809020996, 2.8968050479888916, 0.013297238387167454], [-0.08171039074659348, -0.3115837872028351, -0.24930861592292786, -0.6761887073516846, -0.7197474241256714, 1.5733959674835205, 0.9011762142181396, 2.1760756969451904, -1.626457691192627, 0.6479209065437317, -1.4509843587875366, 0.6879404187202454, 0.17098316550254822, 2.047673225402832, -2.1497344970703125, -2.295376777648926, -0.6845113039016724, 1.745954155921936, 0.6852572560310364, -0.8080608248710632, 0.27187636494636536, 1.9243652820587158, 0.6358205080032349, 0.7211886644363403, -1.4114996194839478, -0.5056453347206116, -0.47564250230789185, 2.3832225799560547, -1.015235424041748, -2.6283388137817383, 2.2722432613372803, -1.7034958600997925, -0.736870527267456, 1.617591142654419, -3.39503812789917, -0.7611901164054871, 0.09567057341337204, 0.6952646970748901, -0.8049864172935486, -0.6893560290336609], [-0.005288288462907076, -0.11745061725378036, -2.227078437805176, -0.0518617257475853, 0.48494434356689453, -0.4039405882358551, -1.2801858186721802, -2.347522735595703, 0.025487016886472702, 0.14204172790050507, -1.29254949092865, -0.9662364721298218, -0.47693514823913574, -0.7769439220428467, -0.9133004546165466, -0.4093535542488098, 0.6349313855171204, 1.6156771183013916, 0.5265929698944092, 2.702324867248535, -0.05261479690670967, -1.1277834177017212, -0.16720017790794373, 0.7557992339134216, -0.5777782797813416, 0.5094491243362427, -1.846851110458374, 0.1713327169418335, -0.257137268781662, -1.4494024515151978, 0.612503170967102, 0.29396340250968933, 0.4675219655036926, -0.12551262974739075, 0.9808569550514221, 0.05437383055686951, 0.7846114635467529, 1.0424182415008545, -0.5499546527862549, -1.4915140867233276], [0.3259298801422119, -0.8193874359130859, -1.3812916278839111, -2.1344528198242188, -0.6293204426765442, 3.22784161567688, 0.7111815214157104, 1.1240330934524536, 0.8533779382705688, -0.43953004479408264, -0.40078791975975037, 0.8903707265853882, -0.5977060794830322, -1.00798761844635, 0.6679940223693848, -0.5839828848838806, 0.5442443490028381, 2.325655460357666, 0.7046141028404236, -1.0257892608642578, 0.9631819725036621, 0.7923223972320557, 0.6554737687110901, -0.6480244994163513, 0.5300537943840027, -0.41663306951522827, 2.32873272895813, 0.5375047326087952, 0.5960090756416321, -1.5483821630477905, -0.12615257501602173, 1.364364504814148, -0.14417189359664917, -0.7789536714553833, -0.9823393821716309, -0.07147571444511414, -0.9322868585586548, 1.5882216691970825, 0.8998818397521973, 0.30702823400497437], [-0.6803854703903198, -0.6285375952720642, -1.6230818033218384, 1.0835142135620117, 2.760465383529663, -0.8181699514389038, -0.9702845811843872, 0.8240513801574707, -0.007767920382320881, -0.3938737213611603, 0.6842957139015198, 0.8793742656707764, -0.49011242389678955, 0.4102781116962433, 0.0005479739047586918, -0.03847537562251091, 1.1039100885391235, 0.2706604301929474, -0.77208012342453, 0.5538274645805359, -1.248254656791687, -1.299408197402954, -1.2450686693191528, 1.6468678712844849, 0.8905779123306274, -0.649715006351471, -0.051228318363428116, 0.9609885811805725, 0.9053069949150085, 1.9654288291931152, 1.1253561973571777, -0.4031203091144562, 1.8701400756835938, 0.3258666396141052, -0.6639975309371948, -0.6330657005310059, 0.1113358810544014, -0.6132940649986267, -1.2467122077941895, -0.22694478929042816], [-0.9269661903381348, 0.8574332594871521, -1.721208930015564, -0.8583329319953918, 0.3607432544231415, 0.776651918888092, -0.003097528824582696, 1.6662806272506714, -1.5246758460998535, 0.1857849359512329, -0.8339715003967285, 1.1572959423065186, -0.22318223118782043, 0.4102158546447754, 0.45777490735054016, 0.33787113428115845, -0.7870861291885376, 0.7037708163261414, -0.7122179865837097, 0.29167234897613525, 1.1903090476989746, -1.8681511878967285, 0.5007021427154541, -1.7820730209350586, -0.5528205037117004, 0.7971556782722473, 1.21772038936615, -0.3517650067806244, 0.17818112671375275, -0.3206701874732971, 1.4434748888015747, 2.913759231567383, 0.158110111951828, 0.7469602227210999, -0.42566928267478943, -1.5812917947769165, -0.11888456344604492, 1.164002776145935, -1.1228667497634888, 1.4154561758041382], [-0.31711524724960327, -0.3945678472518921, 0.9146422147750854, -0.33459654450416565, 0.8134371638298035, 1.0648472309112549, -0.7145712971687317, 1.8827790021896362, 0.9489990472793579, 1.62875497341156, 1.0830986499786377, 2.8933887481689453, 0.20742395520210266, -1.1789261102676392, -0.2182788997888565, -0.3770180642604828, -0.6271462440490723, -1.088459849357605, -0.00016315332322847098, 0.02393442764878273, 0.1697528064250946, 1.2138971090316772, -0.7952215075492859, -0.09885934740304947, -0.4707877039909363, 0.9395369291305542, -1.0030803680419922, -1.6729209423065186, -0.23151828348636627, -1.438175916671753, 0.1333727389574051, 0.8223839998245239, -1.2197922468185425, -1.5109413862228394, -0.26023873686790466, 0.5075422525405884, -0.3000328242778778, 1.6902525424957275, -1.0141453742980957, -0.2814199924468994], [0.011639281176030636, -2.2507612705230713, -2.0896220207214355, 0.5866144299507141, -0.2863987386226654, 0.08383896201848984, -0.49669545888900757, -0.12008397281169891, 0.03825259953737259, 0.7685413956642151, 0.6873101592063904, -1.1339921951293945, 0.42592066526412964, -0.35593804717063904, -0.2719758152961731, -0.6591054797172546, 0.16938327252864838, 1.4994865655899048, 0.8832587599754333, -1.6105021238327026, 1.3925997018814087, 0.32697904109954834, -0.6742410659790039, 1.8467930555343628, 0.2224859893321991, -0.6449863910675049, 0.6093021035194397, 0.8670417070388794, 0.0016592463944107294, 0.20055405795574188, -2.409217357635498, -0.337811142206192, 0.7273898720741272, 0.5078744292259216, -0.8045002222061157, -0.29766765236854553, 2.2176055908203125, 1.1548938751220703, 0.07589764893054962, 1.4084314107894897], [1.2573429346084595, 0.03292402625083923, 0.8617749810218811, 0.1856246292591095, -1.7368457317352295, -0.20303502678871155, -0.9060966372489929, -0.6570031046867371, 2.25834321975708, 1.7315146923065186, 1.324057936668396, 1.4134413003921509, -1.0917080640792847, 0.07188684493303299, 0.597095251083374, -1.6599407196044922, -0.012631234712898731, -1.5900524854660034, 0.5762894749641418, -1.9881072044372559, -0.8280132412910461, 0.2528761625289917, 0.10872777551412582, 2.1318461894989014, -0.5704004168510437, -0.24165824055671692, 0.8506765961647034, 0.4241250455379486, -0.2624751925468445, 0.2834838628768921, 0.6945918798446655, -0.7125536203384399, -0.17724543809890747, -0.6964493989944458, -1.1170320510864258, 2.049988031387329, 1.4313324689865112, 0.2750702500343323, 0.09533156454563141, -1.110045075416565], [-0.8252028226852417, 0.7015536427497864, -0.1621757447719574, 1.7026337385177612, -1.1920526027679443, 0.0598585344851017, -0.5653356313705444, 1.2019431591033936, 0.5753011703491211, 0.47688207030296326, -0.1602754443883896, -0.46803799271583557, -0.8266211152076721, -0.4700762629508972, -1.7367669343948364, 0.1837734431028366, 1.0704436302185059, 0.19676367938518524, -1.0250195264816284, -1.4398313760757446, 1.2612396478652954, 1.4371672868728638, 1.6820111274719238, -1.0472514629364014, 1.1361439228057861, 0.9773238897323608, -0.18777915835380554, -2.319554090499878, -1.2011817693710327, -0.4104352593421936, -0.3943968713283539, 0.6328061819076538, -1.6500325202941895, 0.04703053459525108, -0.8647840619087219, -0.13079321384429932, 0.8298546671867371, 1.0249468088150024, 0.6483073234558105, -0.32587897777557373], [-0.37786099314689636, -0.37776854634284973, 0.4408923387527466, -0.04917069151997566, -1.6797791719436646, -1.4084805250167847, -1.8379223346710205, -0.9550037980079651, 0.8634951710700989, 0.12697449326515198, 1.4341305494308472, 0.47676852345466614, 0.7078108787536621, 1.6029380559921265, -1.9049715995788574, -0.1513565629720688, 1.5373704433441162, 0.1997447907924652, -1.3912519216537476, -1.5320428609848022, 0.8602624535560608, -0.5117448568344116, -0.4727255702018738, 0.5998231768608093, 1.1378079652786255, 0.825171947479248, 0.5857148766517639, 0.8447480797767639, 0.8011094927787781, 1.4870331287384033, -0.9300034046173096, 0.34321683645248413, -0.6818273663520813, 1.7268315553665161, 1.1789675951004028, -0.42090779542922974, -0.23964492976665497, -0.8009340763092041, -1.2791377305984497, 1.2448168992996216], [-2.3467159271240234, -0.89967280626297, 1.4930131435394287, -0.4066280126571655, -1.8078477382659912, -1.1186566352844238, 0.6976015567779541, 0.0485137440264225, -0.6476895213127136, 0.6676397323608398, -0.3325595557689667, 0.33599549531936646, 2.0773472785949707, -0.7452762126922607, -0.11733217537403107, -1.8549809455871582, 0.449843168258667, -0.7682012915611267, 0.9033107161521912, -0.7117712497711182, -0.9827350974082947, -0.5956004858016968, 0.7609031200408936, 0.49998173117637634, 0.25313928723335266, -1.331101417541504, -1.3074371814727783, 1.2119163274765015, 0.4514341354370117, 1.2964316606521606, 1.270469069480896, 0.8709176182746887, 0.42213472723960876, 0.23818686604499817, -1.3225873708724976, 1.2667324542999268, 0.10644537210464478, 0.09240197390317917, -0.053289350122213364, 0.606903612613678], [-0.19772815704345703, -0.37529435753822327, -0.77350252866745, 1.1669073104858398, 1.2282428741455078, 1.4693613052368164, -0.31291839480400085, 1.359455943107605, 0.7462266087532043, 0.5413230061531067, 2.7837815284729004, 0.25782233476638794, 0.6440936326980591, 0.8210939764976501, 0.08512039482593536, 0.2826991081237793, 1.6658902168273926, -1.3303624391555786, 0.0562862828373909, 0.06619760394096375, 0.6002311706542969, -0.17859061062335968, 0.23920634388923645, -0.5704728364944458, 1.877295970916748, 1.0462305545806885, 0.1662246584892273, 0.5151801109313965, 0.5328260064125061, 1.6754413843154907, 0.9169309139251709, 0.0342729277908802, -0.8937336802482605, -1.392160177230835, 0.16771085560321808, 0.4451643228530884, -0.5485290288925171, -0.015711411833763123, -0.35611826181411743, 1.180479884147644], [1.9741053581237793, -1.1013107299804688, -0.6396121978759766, -0.9430173635482788, -1.5624253749847412, -0.08936246484518051, 0.6902081966400146, -0.5114156007766724, -0.5694423913955688, 0.3250288665294647, -1.0181035995483398, -0.19159913063049316, -1.7478811740875244, -0.545059323310852, -0.6767357587814331, -0.23418450355529785, 0.70601487159729, 0.2646292746067047, -0.7361649870872498, 0.18653136491775513, -0.9979093670845032, 1.1692728996276855, -1.3962266445159912, -0.6776124238967896, -1.8229182958602905, -2.3089146614074707, 0.79170823097229, 0.4000738263130188, 0.8482673168182373, -0.46960216760635376, 0.5817858576774597, 1.9548766613006592, -0.6467556357383728, -1.5599743127822876, -0.14536462724208832, 0.4127867519855499, -1.0007350444793701, 1.0167559385299683, -0.5287981629371643, -0.1518854796886444], [-1.746207594871521, 0.2869325876235962, -1.0540369749069214, 1.2361667156219482, -0.7235114574432373, -1.1827698945999146, 0.9718310236930847, 0.37315690517425537, -0.6813827157020569, 0.6876276135444641, 0.9444355964660645, -0.9257885813713074, -1.266873836517334, -0.19405150413513184, 2.355952739715576, 1.1383585929870605, -1.0104931592941284, -1.2319024801254272, -0.556437075138092, 1.2507520914077759, 2.333142042160034, -0.22870036959648132, 0.30516350269317627, 2.4409124851226807, -0.06692749261856079, 0.5635323524475098, -1.520045518875122, 0.0763903558254242, -1.1994330883026123, -0.18887954950332642, -0.3188121020793915, -0.33736592531204224, -1.6137139797210693, -1.3719868659973145, 0.6826868057250977, 1.1587584018707275, 0.861955463886261, 1.3834489583969116, 1.3794749975204468, 0.06826933473348618], [0.266045480966568, -0.40672755241394043, 0.3906422555446625, 1.5916756391525269, -0.6314094662666321, 0.5258514285087585, -0.5887250900268555, 0.07836228609085083, 0.9988270998001099, -0.7256678938865662, -1.0385632514953613, 0.35183385014533997, -0.4159201979637146, 0.17641839385032654, -0.7414048314094543, -2.217078924179077, -0.7866583466529846, 0.45850226283073425, -0.7382892966270447, 1.016315221786499, -1.200487732887268, -0.5107527375221252, 0.19625359773635864, -0.04550603777170181, 1.8095899820327759, -0.6494185924530029, -0.37905752658843994, 0.251148521900177, -0.8942086696624756, 0.563960075378418, 0.10996963083744049, -1.0058679580688477, -0.4998018145561218, 0.03252294287085533, 2.175379991531372, -1.9196544885635376, 0.3089117109775543, -0.3204408884048462, -0.4255369305610657, 0.42002713680267334], [0.10814888775348663, -0.7053077816963196, 0.44690534472465515, 1.2018886804580688, -0.8151742815971375, 0.002195779699832201, -0.2755766212940216, 0.8594473004341125, -0.11461589485406876, -1.483203411102295, 1.2276214361190796, 0.09872408956289291, -0.5426802635192871, 1.337861180305481, -0.40985846519470215, -1.2169691324234009, -1.8097022771835327, -2.481233596801758, -1.0604864358901978, -0.22559309005737305, -0.015534553676843643, 0.34971562027931213, 1.179778814315796, 0.5106011033058167, -0.7313194274902344, 1.0392171144485474, 0.5600759983062744, 0.019339947029948235, -0.9191494584083557, -0.5777679085731506, 1.2690608501434326, 0.659433126449585, 0.7561657428741455, 1.150054931640625, 0.680401623249054, -0.22562341392040253, -0.8027506470680237, -0.11174366623163223, 0.420719712972641, 0.25033363699913025], [0.9835149049758911, 0.4392143189907074, 0.4893026053905487, -0.6769135594367981, -1.3620315790176392, -0.4884621798992157, -0.17290139198303223, -0.7001248598098755, 0.11255796253681183, -0.6591413021087646, 0.3691927194595337, 0.03794560953974724, -1.5255424976348877, -0.0007009884575381875, -0.05121881514787674, 1.2816801071166992, 0.9819318652153015, -1.6148779392242432, -1.95150625705719, 0.30311864614486694, -0.5656529664993286, -0.3108240067958832, -0.3645506799221039, 0.7053578495979309, 0.5209680795669556, 0.7527819871902466, -1.3426547050476074, 0.45947882533073425, -0.08678299188613892, 1.4627084732055664, 0.12693342566490173, -0.6411941647529602, -0.6100248694419861, 0.024947065860033035, 1.2137198448181152, 1.9694795608520508, -0.23039335012435913, 0.33343443274497986, 0.4855600893497467, -0.047316085547208786], [1.2875633239746094, 1.4593300819396973, 1.32845938205719, -1.07611882686615, -0.5778303146362305, -1.6038955450057983, -0.4056766629219055, 0.6647968292236328, -0.8440065979957581, 1.1202468872070312, -0.4330425560474396, -0.4448608458042145, 1.5859450101852417, -0.9712733626365662, 0.5108254551887512, 1.0876753330230713, 0.3850197494029999, -0.359172523021698, 0.3149486482143402, -1.0163217782974243, 0.0850752666592598, 1.1531009674072266, -1.056143879890442, 1.6202129125595093, 0.10633084923028946, -0.9252175688743591, -0.5966329574584961, 0.02428862825036049, 0.2812671661376953, 1.6147511005401611, -0.7612103223800659, 0.25464531779289246, -1.561920404434204, -0.16458913683891296, 0.1885717809200287, -1.3355966806411743, 0.9688978791236877, 0.1918962001800537, -1.7760497331619263, 0.509101927280426], [-0.13752064108848572, -0.420665979385376, 0.48272913694381714, 0.21516749262809753, 0.6266766786575317, 0.8825502395629883, -0.34759119153022766, -1.062074065208435, -0.9158816337585449, -1.336627721786499, 0.48321881890296936, 0.3745994567871094, -0.7685021162033081, -0.7763913869857788, 0.16952085494995117, 1.3199758529663086, 0.15768279135227203, -1.0735783576965332, 0.34709131717681885, 0.4519064724445343, 0.7106276154518127, 1.218507170677185, 0.045364268124103546, 0.4486944079399109, -0.23282618820667267, 0.4286803901195526, -1.1621932983398438, 1.0932468175888062, 0.37352195382118225, -1.4718222618103027, 0.40937602519989014, 0.8361247777938843, -0.6777126789093018, -0.232865571975708, -0.6232704520225525, -0.900428295135498, -0.12813721597194672, 1.9845877885818481, 0.9234973192214966, 0.11512645334005356], [-1.1705076694488525, 1.5464344024658203, -1.7878307104110718, 0.8252027630805969, 0.5443540215492249, 0.08418532460927963, -0.2627891004085541, -0.13102006912231445, 0.5880568623542786, 0.22540636360645294, -0.5776234865188599, 1.0330077409744263, -1.413706660270691, -0.7175799608230591, 0.6384310126304626, -1.2633404731750488, 0.557982861995697, -0.5455490350723267, 0.5765182971954346, -1.4672380685806274, -0.5109933018684387, 2.2136285305023193, -2.467635154724121, 1.0936169624328613, 0.7182426452636719, -0.17238172888755798, 2.130030870437622, 2.4355287551879883, 1.2241250276565552, -1.4386894702911377, 0.7376496195793152, -0.5134938955307007, 0.2883242964744568, 0.1148591861128807, 0.47709721326828003, -0.5775479674339294, 0.30226531624794006, -0.9262314438819885, 0.05371065065264702, -0.21276822686195374], [0.1911221593618393, 0.9480357766151428, 0.7464298009872437, 1.3525099754333496, -0.2949315011501312, -0.10593310743570328, -1.2316875457763672, -2.1342546939849854, 0.268785297870636, 0.3664664626121521, -1.3443018198013306, 0.5739619731903076, 0.5264419317245483, 0.9223672151565552, -0.31409114599227905, 0.16839610040187836, 1.3225525617599487, 2.2811343669891357, -0.979479968547821, -0.2681909501552582, -1.4100430011749268, 2.1587860584259033, -0.9093277454376221, -0.1406826227903366, 0.6512392163276672, 0.14752641320228577, -1.0015146732330322, 0.711926281452179, -0.9260438680648804, -0.1986706256866455, 0.7490365505218506, -1.9391456842422485, 0.7421678900718689, 1.483652949333191, 0.9887404441833496, -1.6342685222625732, 1.2376956939697266, -0.933923602104187, -1.0373473167419434, -1.0313334465026855], [0.4498899579048157, -1.2691044807434082, 1.028594732284546, 0.14120575785636902, 0.7160324454307556, -0.23163636028766632, -1.128616213798523, 1.191148281097412, 0.25477632880210876, -0.24873201549053192, -1.425815463066101, 0.46284446120262146, -0.350911945104599, 0.5547111630439758, 0.3255026042461395, -1.4185477495193481, 1.3325432538986206, 0.7612640857696533, -0.10925603657960892, -1.5494623184204102, 0.03287225961685181, 0.3740517199039459, 0.7411414980888367, -0.19135800004005432, 1.2790414094924927, -0.582500159740448, -0.6076816320419312, -0.5789722204208374, -1.165140986442566, 1.217235803604126, 0.15023274719715118, 0.06509491801261902, 1.569656491279602, -0.43254318833351135, -2.852912425994873, -0.08257756382226944, 0.5335316061973572, 0.3451811671257019, -0.18904505670070648, 1.5258395671844482], [-0.8807867765426636, -1.3002451658248901, 0.21669219434261322, 0.5362619757652283, 0.16649439930915833, -1.5253459215164185, 0.3662321865558624, -0.02409488707780838, 1.7433264255523682, -1.9442026615142822, 1.5445586442947388, 0.7529739737510681, 0.6535558700561523, 0.502126157283783, 0.29748067259788513, -0.3240319788455963, 0.23847942054271698, -1.0994974374771118, 0.40641331672668457, 1.3207805156707764, 0.0822892114520073, -2.0414514541625977, 1.3239976167678833, 0.8753232359886169, 1.0330463647842407, 0.14501672983169556, 2.30570125579834, 0.5010163187980652, 1.5406386852264404, 0.23746556043624878, 2.3149125576019287, 0.9733021259307861, 1.2437214851379395, 0.32718896865844727, -0.05445632338523865, 0.1789502501487732, 2.1361072063446045, -0.06580017507076263, -0.6688298583030701, 0.4583441913127899], [0.9316183924674988, 0.9838689565658569, -0.40404146909713745, 0.20124492049217224, -0.7188757061958313, 1.288514256477356, 0.7800561189651489, -0.5323683619499207, 1.6717435121536255, -0.647047221660614, 1.5291433334350586, -0.08413082361221313, -2.2544047832489014, 1.8975528478622437, 0.13662634789943695, -1.2720705270767212, 1.8622878789901733, -0.3200923502445221, 2.0209810733795166, -1.595483422279358, 0.596011757850647, -0.7308835387229919, -0.1747242510318756, -1.028208613395691, -0.11941051483154297, -1.7158308029174805, -0.4847795367240906, -0.7643048167228699, 0.2244510054588318, -0.7209122180938721, -0.8279290199279785, -0.42063549160957336, -0.2013154923915863, -0.13966412842273712, 0.34305891394615173, 0.6289693713188171, -0.18375293910503387, -0.0655163824558258, 2.3769469261169434, -0.6007442474365234], [-0.3001914322376251, 0.024212239310145378, -0.6834836602210999, -0.8728731274604797, -1.293789267539978, -0.6328346133232117, -0.13479316234588623, 0.06959271430969238, 1.1515690088272095, 0.5704413652420044, -1.3150875568389893, -0.42444372177124023, -0.24734088778495789, -0.13936880230903625, -1.4697985649108887, -1.2743768692016602, 1.0255639553070068, -0.41500040888786316, 0.411650687456131, 0.48945093154907227, 0.4657604992389679, -0.6305304765701294, -0.49908846616744995, -0.5351296067237854, -0.599372923374176, 0.8577568531036377, -0.15486612915992737, -0.6911613345146179, -1.032593011856079, 0.29272446036338806, 0.897872269153595, -1.192797064781189, -1.0040332078933716, 0.1538155972957611, 1.710864543914795, 2.2510740756988525, -0.08419553935527802, 1.0070829391479492, -0.76956707239151, -0.276804655790329], [1.2367740869522095, -0.24400806427001953, 0.8672477602958679, 0.5900530219078064, 0.544227123260498, -0.5256696939468384, 0.383775532245636, 0.9904181957244873, -0.8717327117919922, -0.6171626448631287, -0.1685216873884201, -0.6102948188781738, 0.43223416805267334, 0.15712018311023712, 1.1705000400543213, 0.18986748158931732, 0.5591993927955627, 0.40123680233955383, 0.318947970867157, -1.0776463747024536, -0.4282601773738861, -0.7945091724395752, -0.2871097922325134, -0.8659905195236206, -0.047410834580659866, -1.9286013841629028, -0.2932887673377991, 1.808231234550476, 0.9884967803955078, 0.37677425146102905, -0.06391310691833496, -0.41456472873687744, 0.27747905254364014, 0.9972524642944336, -0.6009415984153748, -0.45203232765197754, -0.2413347065448761, -1.328493356704712, 0.6470511555671692, -0.4586179852485657], [-0.6741765141487122, 0.3034978210926056, -0.05860589072108269, -0.16505354642868042, -2.4620325565338135, -0.13637888431549072, -0.5557175874710083, -2.8824055194854736, 0.49823305010795593, 0.8112021088600159, 2.3436059951782227, -0.06300702691078186, -2.009629964828491, 0.5097329616546631, 0.3998425006866455, 0.22467193007469177, 0.8951475024223328, -0.03355928882956505, -0.048263125121593475, 0.9005342125892639, 0.3442661762237549, 0.029812628403306007, 1.4855296611785889, 0.19471991062164307, -0.6115192770957947, 0.32916104793548584, -1.2836683988571167, -0.9262354373931885, 1.2008788585662842, 0.23993325233459473, -0.6209231019020081, -0.7714937925338745, -0.019808506593108177, 1.6751846075057983, -2.2039952278137207, -1.6851142644882202, -0.9692484736442566, -2.0439395904541016, -1.2762773036956787, 0.6903734803199768], [0.7974774837493896, 1.6496986150741577, -1.5651538372039795, 1.898746371269226, -2.898310899734497, 1.1106526851654053, 0.8119369149208069, 0.8597769141197205, 0.1118406280875206, -0.7052208781242371, 0.2908845841884613, 1.747746467590332, -0.23699738085269928, 0.77224200963974, -0.4945639967918396, -0.6091305017471313, 0.18730774521827698, -1.685627818107605, 0.3230462670326233, -0.13537712395191193, 0.06814856082201004, 0.5919156670570374, -1.5895287990570068, 0.06459562480449677, 0.7352768778800964, 1.169214129447937, -1.0274372100830078, 0.45515742897987366, 1.24958074092865, -0.05803654342889786, -1.1723673343658447, -1.601087212562561, 1.756990909576416, -1.6776424646377563, 0.8638888597488403, -0.874518871307373, -0.9738816618919373, 0.3589828908443451, 1.6700814962387085, -1.990725040435791], [-0.5241962671279907, -1.8642600774765015, 0.6719985604286194, -0.28895246982574463, 0.5538302659988403, 2.004744291305542, -0.13444741070270538, 0.938604474067688, 1.1672793626785278, 0.8914728164672852, -1.5640761852264404, 0.20207829773426056, -0.3404775559902191, -0.8924145102500916, -0.23785437643527985, 0.4706341624259949, -0.3990691900253296, 0.11827286332845688, -1.2745652198791504, -0.6473382115364075, 1.3270314931869507, 1.4901460409164429, -0.6142864227294922, 0.8203825354576111, -0.4619285762310028, 0.2743145227432251, 1.5807712078094482, -1.197059154510498, -0.9544451832771301, 0.756893515586853, -0.4063081741333008, -0.5197827219963074, 0.9917694330215454, 0.14642804861068726, -1.575104832649231, 0.6787887215614319, -0.3187006413936615, 0.47410276532173157, -1.455365777015686, 0.3609853982925415], [1.2684458494186401, 0.8807798027992249, 0.712095320224762, -0.9995734095573425, 0.3095805048942566, 0.04197731614112854, -0.4114684462547302, 0.15848717093467712, -0.49938568472862244, 2.1516926288604736, 0.5668363571166992, 0.3941448926925659, -0.1049254983663559, 0.6419390439987183, 2.0924947261810303, 1.7624363899230957, -0.5546153783798218, -1.264182448387146, -0.15612757205963135, 0.22733105719089508, 1.3855382204055786, -1.822838306427002, -0.6325498819351196, -0.6081002950668335, -2.5066637992858887, -0.22197866439819336, 1.333971619606018, -2.1850459575653076, -0.7580808401107788, -0.7242244482040405, -0.45967739820480347, 0.4324147403240204, 0.8259572386741638, -0.8909503817558289, 0.2812087833881378, -1.324220895767212, -0.5250368714332581, 0.49221178889274597, 0.37401512265205383, -1.5158313512802124], [0.9916895031929016, -0.3890325725078583, 0.6429494023323059, -1.2246074676513672, 1.4873207807540894, -1.0841768980026245, 0.45934051275253296, -1.766201376914978, -1.0275180339813232, 1.7872517108917236, 2.136914014816284, -0.07120727002620697, -2.015831708908081, -0.12637309730052948, -0.2083783596754074, -0.7601779103279114, -0.442313551902771, -0.1265283226966858, 1.087733268737793, -0.12298596650362015, 1.1095569133758545, 1.2806789875030518, -0.575048565864563, 1.6979265213012695, 1.3073045015335083, -1.3991971015930176, 0.023945940658450127, 1.0540730953216553, -0.6481223702430725, 0.7527980804443359, -0.3062135875225067, 0.3964514136314392, 2.578176736831665, -0.2551057040691376, 0.9198501110076904, 1.7221812009811401, 1.5624788999557495, 1.433708667755127, -1.1879677772521973, -0.490746408700943], [0.8073734045028687, 0.47562211751937866, 1.1413222551345825, -0.46857741475105286, -0.35527369379997253, -0.7380509972572327, -0.8825297951698303, -1.1370002031326294, 0.06925441324710846, 2.346674680709839, -0.6216861605644226, 0.6983975768089294, 1.9867026805877686, 1.80402410030365, -1.6078369617462158, -0.5921781659126282, 0.30468103289604187, 0.7329971790313721, -0.6011906862258911, 0.141028493642807, -0.5975088477134705, 1.2141073942184448, -1.4789655208587646, -2.2265167236328125, -0.3132372200489044, 0.30432382225990295, -1.4676753282546997, 0.38436099886894226, -0.7671728730201721, -0.016997825354337692, 0.5237131714820862, -0.4142944812774658, -2.150780439376831, -1.0787270069122314, 0.13627447187900543, -0.7998726963996887, -0.14374330639839172, 0.05852692201733589, -0.5351091623306274, 0.04223403334617615], [-1.1002715826034546, 1.3388283252716064, 0.24543368816375732, 1.206868290901184, -0.3389270305633545, -0.6870833039283752, -0.8351108431816101, 0.39390549063682556, 1.2366108894348145, 0.2976408898830414, 0.9296383857727051, 0.258705735206604, -0.4679904878139496, 1.9595379829406738, 1.8587431907653809, -0.3102656304836273, 1.0751029253005981, 0.5957848429679871, -0.30363938212394714, -0.7053697109222412, 0.36420926451683044, -0.088874451816082, 0.46917545795440674, 1.9933676719665527, -0.9545634388923645, 0.4003056287765503, -0.3637070953845978, -0.5883966684341431, -0.4518916606903076, -0.12307559698820114, -0.5557323098182678, -1.6720343828201294, 1.2740424871444702, -0.8698726296424866, 0.968342661857605, -2.089568614959717, 0.46996429562568665, 0.5260499119758606, -0.5729385614395142, 1.9250849485397339], [-1.0397400856018066, -0.25835132598876953, -0.7700251340866089, 0.5281955599784851, 1.8073064088821411, -0.0419766791164875, 0.5400679111480713, -1.2651419639587402, -1.3282781839370728, -0.7383158206939697, 0.6607035994529724, -0.11772540956735611, -0.560149610042572, 1.064048409461975, 1.7228646278381348, 0.7718058824539185, -0.357928603887558, -0.00661329785361886, 0.39543619751930237, -0.9205941557884216, 0.8742279410362244, -0.5847007036209106, -0.8057700991630554, -2.061549425125122, 0.8006914258003235, 1.9507567882537842, -0.6561373472213745, -1.5908265113830566, -0.11403249204158783, 0.172145813703537, 1.4790629148483276, 0.8523200154304504, 0.7865114808082581, 1.7840877771377563, -0.4090098440647125, 1.4738624095916748, -0.7002755403518677, 2.5485198497772217, 1.1087809801101685, -1.7982397079467773], [-0.26530924439430237, 0.5926434993743896, -0.764144241809845, -2.1634087562561035, -1.7778503894805908, -1.1227136850357056, -0.9814715385437012, -0.28531116247177124, 0.1349181830883026, -1.2150019407272339, -0.47500619292259216, -0.17573489248752594, -2.1026949882507324, 0.4414103031158447, 1.360914945602417, 0.45577308535575867, -1.395119547843933, 0.319510281085968, 0.1873033493757248, -1.0759532451629639, 1.970114827156067, -0.07322341948747635, 0.5828496813774109, -1.3406243324279785, -0.07947667688131332, -1.375953197479248, -0.1964063048362732, -0.4447193741798401, -0.35794776678085327, 0.5178666114807129, 1.2960203886032104, -0.28330785036087036, 1.8836697340011597, 0.3662146031856537, -1.419184684753418, 0.36825019121170044, 0.878119945526123, -0.059740036725997925, -0.8222374320030212, 0.036123789846897125], [0.7636153101921082, -0.35128188133239746, 0.755494236946106, 0.6389200687408447, 1.0173460245132446, -0.6519820690155029, -0.29930436611175537, -0.01604953408241272, 1.3453108072280884, -1.2279516458511353, 0.882645308971405, 0.31420010328292847, -0.22884264588356018, 0.1509200483560562, 0.26004311442375183, -0.5189947485923767, 0.33096444606781006, 1.8945902585983276, 0.8053922057151794, 0.16050028800964355, -1.9407485723495483, 0.2291242778301239, 0.06706348061561584, -0.5148977041244507, 0.600650429725647, 1.2572276592254639, 0.08621761947870255, 0.965467631816864, 0.6427714824676514, 0.2662772834300995, 0.264265775680542, 0.06782350689172745, 0.5137426853179932, -0.8294804096221924, 0.4423307776451111, -1.4370259046554565, 1.2588515281677246, -0.8784453272819519, -0.16467496752738953, -1.0206844806671143], [-0.3567814826965332, -1.4822065830230713, -0.8197727799415588, -1.2283281087875366, 0.24156974256038666, 1.0877540111541748, 1.611005425453186, -1.262636423110962, -0.309282511472702, -0.2528930604457855, 1.0437307357788086, 0.20479902625083923, -1.1693466901779175, 1.577527642250061, 1.5230281352996826, -1.0142126083374023, 0.6038170456886292, 0.7046084403991699, -0.318377286195755, -0.12322422862052917, -1.4615075588226318, -0.253794401884079, 0.04529830813407898, 0.3196573853492737, 1.101677656173706, 0.1941816210746765, -0.6940388083457947, 0.0001513083407189697, -0.1954461634159088, -0.5883099436759949, 1.4571428298950195, -0.9786142706871033, -0.3419259488582611, -1.0326378345489502, -0.6426025629043579, 0.8755779266357422, 1.0665043592453003, -0.13857071101665497, -1.795283317565918, 0.620603621006012], [-0.07340056449174881, 0.1348968744277954, -0.3611682653427124, -0.9848730564117432, -0.4920148551464081, -0.2673652172088623, -0.6646419763565063, -1.1105067729949951, 0.7133786082267761, 0.4919402599334717, -1.5866011381149292, 0.9688788652420044, -0.28564968705177307, 0.04062316194176674, 0.32639679312705994, 0.28671908378601074, -1.2348017692565918, 1.2252155542373657, 0.08856198191642761, -0.4930402934551239, -1.6131778955459595, 0.40965884923934937, -1.106081247329712, -0.11548329889774323, 1.6093249320983887, 0.03282839432358742, -0.2875494658946991, 0.2727927565574646, -0.25633931159973145, 0.5118998885154724, 0.5683027505874634, -0.7685091495513916, -0.8824706077575684, -0.6205592751502991, -0.8885642886161804, 1.074463963508606, 0.5851582288742065, 1.0772298574447632, 0.17123498022556305, 0.2698621451854706], [-0.9580910205841064, -0.3635770380496979, 1.6847994327545166, -0.7214136719703674, -0.40594854950904846, 0.16255827248096466, -1.2187519073486328, 1.0587035417556763, -1.7378227710723877, 0.24924927949905396, 1.8596152067184448, 1.0194766521453857, -0.002861456014215946, 0.20195145905017853, 1.614684820175171, 0.49143797159194946, -0.3246212601661682, -0.0929863229393959, -1.2471559047698975, -0.19079574942588806, -0.06034087389707565, 1.6154005527496338, 0.408133864402771, 1.4074554443359375, 0.05710763856768608, -1.3588528633117676, -1.2882881164550781, 0.5402615666389465, -0.7565352916717529, -1.2085692882537842, 1.3152035474777222, 0.9806469678878784, -0.6783950328826904, 0.18906381726264954, -1.7122938632965088, 1.7985129356384277, -1.3443466424942017, 2.484342575073242, -0.9351086616516113, -0.5894517302513123], [-1.832466721534729, -0.6068822741508484, -0.019810587167739868, -2.515594005584717, 0.22838200628757477, 2.6074182987213135, -0.6913626194000244, 1.4054542779922485, -0.9201732277870178, 1.3878618478775024, -0.7672152519226074, 0.056077990680933, -0.7716696858406067, -0.714354932308197, 0.07767041772603989, -0.2869666814804077, 0.7882885336875916, 0.2485497146844864, 0.3663155734539032, -1.7969626188278198, 0.4905013144016266, 1.2798492908477783, -0.5821321606636047, -0.5048606991767883, 0.6602979302406311, 2.4419620037078857, -0.623623251914978, 0.2048528790473938, 1.3699873685836792, 1.0266448259353638, -1.4794702529907227, 0.681561291217804, -1.3067024946212769, 0.6354679465293884, 0.060681313276290894, 0.6866368651390076, -0.4317662715911865, -1.3080289363861084, 1.1267825365066528, -1.3823341131210327], [2.1781704425811768, -0.06408818811178207, -1.2852646112442017, -0.7233055830001831, 0.7956066727638245, -0.7896400094032288, 1.7529182434082031, -2.5749049186706543, -1.3936779499053955, -0.011260072700679302, 1.844026803970337, -0.5286062359809875, 2.3727126121520996, -0.2893022894859314, -0.565475583076477, 0.44801071286201477, 0.5793012976646423, -1.33014714717865, -1.2166719436645508, 0.4299076199531555, -0.4323541820049286, -0.38506996631622314, -2.2716360092163086, 0.4256415367126465, -0.29180240631103516, -0.3264065384864807, -1.456348180770874, 0.179622620344162, 0.6857800483703613, -2.335892677307129, 0.5594527721405029, 1.3148690462112427, -1.0224039554595947, 0.46021077036857605, -0.38605839014053345, 0.15515196323394775, 1.0016916990280151, 0.30548274517059326, -1.2860057353973389, 0.1844923049211502], [-0.27850034832954407, 0.8363150358200073, -0.5662693381309509, -0.9717121124267578, -1.1684023141860962, -1.8423134088516235, -1.7780019044876099, -2.160963296890259, -1.143146276473999, 0.05476712808012962, -0.7349840998649597, 0.43960240483283997, 0.8173909187316895, -0.23202531039714813, -1.0274765491485596, 0.776390552520752, -0.8443471193313599, 0.08756197243928909, -0.6206347942352295, -0.9198800921440125, -1.3099900484085083, -0.9651615619659424, 0.12971651554107666, -1.0330138206481934, 1.215075969696045, 3.3825933933258057, -0.9078055620193481, 1.909914493560791, 0.593352735042572, -1.2872248888015747, -0.3354664742946625, 0.7804646492004395, 0.560151994228363, -1.6529948711395264, 0.598203718662262, 0.6493488550186157, -1.121598482131958, -0.5910808444023132, 0.7722518444061279, -0.8291086554527283], [-0.29880064725875854, -2.6845786571502686, -1.243808388710022, -0.024686643853783607, -1.2883226871490479, -0.3798021376132965, 0.18078996241092682, -1.3296751976013184, 1.1509968042373657, 0.22534483671188354, -0.06382226943969727, -0.15063965320587158, -0.7239401340484619, -0.730911374092102, 1.3826241493225098, 0.6977787017822266, 0.8438374400138855, 0.4794427752494812, -0.8662155866622925, -1.1796741485595703, -0.5750607848167419, -0.4511614441871643, -1.008934497833252, 0.7611655592918396, -0.4985179305076599, 1.0067448616027832, -1.9369585514068604, 0.585520327091217, 0.8629401922225952, 0.5360696911811829, 1.6071783304214478, 0.7396556735038757, 0.4252535402774811, 0.677422285079956, 0.17551776766777039, -0.22048458456993103, -1.06925630569458, -1.232364535331726, 0.5009337067604065, 1.120421290397644], [0.47278839349746704, 0.4235613942146301, -0.07955826073884964, -0.12480480968952179, -0.4007762670516968, -0.1345854252576828, 0.5220991969108582, 0.011616821400821209, -1.27629554271698, -0.6407543420791626, 0.5729352235794067, 0.3382924497127533, -0.4896851181983948, 0.4209223687648773, 0.3144756853580475, 1.7742094993591309, -0.8267821073532104, -0.6158722043037415, 0.8547406196594238, -1.9525667428970337, -0.5894072651863098, -0.28309211134910583, 0.5540493130683899, 0.42932379245758057, -0.2480352222919464, -0.7915875911712646, -0.3706440329551697, -2.208198070526123, -1.0779474973678589, -0.22416344285011292, 1.4767680168151855, -0.8550183773040771, -1.7252647876739502, -0.303048312664032, -1.0415971279144287, 1.2476998567581177, 0.6518465280532837, 0.36740386486053467, -0.3515930473804474, 0.4354027509689331], [0.23778197169303894, 1.2216041088104248, 1.5769963264465332, 1.3808720111846924, 0.9693081378936768, -0.1465381234884262, -0.284264475107193, 1.433472752571106, -0.5489039421081543, -1.026482343673706, 0.45865675806999207, -1.547401785850525, 1.0863306522369385, 1.073081135749817, -0.7787511348724365, -0.5186028480529785, 0.4396168291568756, -0.015292410738766193, -1.1989017724990845, 2.133033037185669, 2.346937656402588, -0.8860450983047485, 0.5549333691596985, 1.06270170211792, -0.5283611416816711, -0.1647460162639618, 1.294019103050232, -1.2811516523361206, 0.5946880578994751, 0.9615371823310852, -0.15906734764575958, 0.3130137622356415, -1.3409026861190796, 0.09083089977502823, -0.11052357405424118, 0.2051834911108017, 0.7001256346702576, 1.7410460710525513, 2.661067008972168, 0.7976023554801941], [-0.39576393365859985, -0.9609776139259338, 0.7832814455032349, 0.6243661046028137, -0.5536527037620544, 1.0151944160461426, -0.7307382225990295, 0.9048829078674316, 0.9583379030227661, -0.5804072618484497, -1.2712159156799316, -1.4942148923873901, -0.3596561551094055, 0.6753538846969604, 0.458629310131073, 0.2511173188686371, -0.8282133936882019, -0.2583341598510742, 0.3596581220626831, -0.2757469117641449, -0.5736466646194458, 0.32649463415145874, 0.18488669395446777, 1.5987564325332642, 0.14946790039539337, 1.1390241384506226, -1.6014809608459473, 0.39015716314315796, -0.1742161214351654, -1.6994733810424805, 2.345334529876709, -0.05784941464662552, -0.27986636757850647, -1.9123932123184204, -0.23472540080547333, -1.0557862520217896, -0.19028660655021667, -2.1294219493865967, -1.7529668807983398, -0.2528632879257202], [1.225630760192871, -1.8510329723358154, 0.910912811756134, -1.7805136442184448, -1.7411099672317505, -0.17435134947299957, 1.045564889907837, 2.0428621768951416, -0.7479397058486938, 0.5622762441635132, -0.10100045055150986, 0.9888290762901306, -2.3582775592803955, 0.5298491716384888, 0.5027872920036316, 1.6359962224960327, -0.36300280690193176, -1.4445202350616455, -1.9022382497787476, -0.6661268472671509, -0.07058033347129822, -2.3065054416656494, 1.1222909688949585, -0.9581277966499329, -0.926621675491333, 0.21830017864704132, -0.7716604471206665, 0.18668819963932037, 0.022724982351064682, -1.5037192106246948, 0.9606279134750366, -0.7647169232368469, 0.3582281768321991, -0.2381376326084137, 0.9801253080368042, 1.6774423122406006, 0.5659725069999695, 0.029158011078834534, -0.8260034918785095, -0.5288829803466797], [1.6129549741744995, 0.6892915964126587, 0.2754419147968292, -1.326292872428894, -0.1700594574213028, -0.5782964825630188, 0.4949903190135956, 1.143571376800537, -0.3143223524093628, 0.9767487049102783, 1.3156664371490479, 0.6120603680610657, -1.2869664430618286, -0.7054400444030762, 0.516304612159729, -1.9103838205337524, 1.2927591800689697, 1.5316745042800903, 0.26120370626449585, 0.1686645746231079, -1.9808169603347778, -0.5390315055847168, -0.10482342541217804, -1.4937602281570435, -0.34319907426834106, 0.5512987971305847, -0.4849696159362793, -0.14749038219451904, 0.5461867451667786, -0.868554949760437, 2.0303783416748047, -1.5884767770767212, 0.44274047017097473, -0.3655913472175598, -0.01406041532754898, -0.2815879285335541, 0.9429219365119934, -0.9380174279212952, -0.1707674115896225, 0.23159398138523102], [0.8476254940032959, -0.13234007358551025, 0.14705726504325867, 0.6493425965309143, -3.1531479358673096, 1.1422070264816284, 2.1013519763946533, -0.28938835859298706, 1.0818748474121094, 1.4278672933578491, -1.0663293600082397, 2.585592746734619, -0.7678201198577881, 0.14939048886299133, 0.4394075274467468, 1.0317398309707642, 0.8499947786331177, -0.024029793217778206, 1.6959428787231445, -1.5555685758590698, 2.7586252689361572, 0.17390333116054535, -0.4212154746055603, -0.199191153049469, -1.2888953685760498, 0.9309492707252502, 1.8480478525161743, -0.23419225215911865, 0.008004778996109962, 0.17027099430561066, 1.9737272262573242, -1.606856346130371, -0.621502697467804, 0.3020614683628082, 0.6076458096504211, -1.1623083353042603, 0.4498942196369171, -0.2622518539428711, -0.6936811804771423, -1.3684453964233398], [-0.05772373452782631, -0.8908382058143616, -0.3969658315181732, 0.8271538019180298, 0.029216406866908073, -0.2181038111448288, -0.9799907803535461, -0.9855098128318787, -0.387727826833725, -0.29110196232795715, -1.6337242126464844, 0.4145917296409607, -0.09226998686790466, -0.36413851380348206, 0.6446906924247742, 1.1921454668045044, -0.014786995947360992, 1.554006576538086, 0.36317798495292664, -0.9693796038627625, 0.866738498210907, -0.02827397733926773, -0.6695818901062012, 1.6965481042861938, -1.690667748451233, -0.7855018973350525, 1.6302567720413208, -0.4311105012893677, -0.22349727153778076, 0.22029760479927063, 1.9405454397201538, -0.40590059757232666, -0.31605976819992065, 1.9722706079483032, 0.9048154950141907, -0.8499410152435303, -0.6444416642189026, -1.3776633739471436, 0.45608216524124146, 1.0229908227920532], [-0.710757315158844, 0.07059495151042938, -0.7291157841682434, 1.1266025304794312, -1.4723310470581055, -0.3043365478515625, -1.5701498985290527, 0.8766537308692932, -0.5613874793052673, 2.240370273590088, -2.854210138320923, 0.12141577899456024, 1.5232112407684326, -0.9338489770889282, 0.5373321175575256, 0.08410706371068954, 0.905896008014679, 1.4436269998550415, 0.5712393522262573, 0.27472764253616333, 1.1083776950836182, -0.6714447140693665, 0.2086978703737259, -0.18512728810310364, -1.6165841817855835, 0.13439086079597473, 1.3865697383880615, 2.488781213760376, -1.80100417137146, -1.8399771451950073, -0.26811841130256653, -0.277942419052124, -0.1453751176595688, 0.1981545388698578, 1.3336058855056763, 1.483149528503418, -0.9841390252113342, -1.4998997449874878, 0.7014046907424927, -2.2432966232299805], [0.9041302800178528, -0.3311240077018738, 0.5689223408699036, 0.5089585781097412, -0.4260607659816742, 0.005852338392287493, 0.5843046307563782, -0.4102802276611328, -0.26484984159469604, 0.808595597743988, 0.6520991325378418, 0.8914278149604797, -1.083003044128418, 1.8135086297988892, 0.017084652557969093, 1.2919021844863892, -0.5934523940086365, -0.2538334131240845, 0.8390635848045349, 0.9672210216522217, -0.5207569599151611, 0.009697441011667252, 1.9245070219039917, 0.36322304606437683, -0.22061166167259216, -0.40652546286582947, -1.5468430519104004, 0.04205683246254921, -0.38083532452583313, 0.37827277183532715, -1.2365007400512695, 1.2492806911468506, 0.10715563595294952, -0.8190471529960632, 0.03860081732273102, 0.3398863971233368, -0.44464486837387085, -0.010656310245394707, 0.37446826696395874, -0.13047856092453003], [-0.6802408695220947, 0.5551729798316956, -1.794044852256775, -1.2652690410614014, -0.0004924232489429414, -0.22365429997444153, -0.9214509129524231, -0.07445056736469269, 0.8214294910430908, -0.1771586537361145, -1.1539226770401, -1.223209261894226, -0.6556462645530701, -1.0181028842926025, -0.7558273673057556, 0.21532367169857025, 0.4957142472267151, -0.9842696785926819, 3.44956111907959, 0.1998257040977478, 0.09953044354915619, 0.4093981385231018, 0.7788167595863342, 0.4053390920162201, 0.2991247773170471, 0.6586197018623352, -0.711137592792511, 0.0749860554933548, 2.3230936527252197, -0.5822020173072815, 1.2437770366668701, 1.401041865348816, -0.716517448425293, 0.18511135876178741, -0.5820023417472839, -0.48514771461486816, -0.8797940611839294, -1.1603294610977173, 0.2229836881160736, 0.14140422642230988], [-1.9142563343048096, 1.4917267560958862, 0.7305096983909607, 1.221678614616394, 0.6256528496742249, -0.0815766453742981, -0.26477286219596863, -0.20979224145412445, 0.09965137392282486, -0.9504868388175964, 0.26064205169677734, 0.7292882204055786, -0.7366876006126404, 0.3018716871738434, 0.049234915524721146, -1.7352036237716675, -1.3072044849395752, -2.7050116062164307, 0.2525602877140045, 0.7639905214309692, -0.9279879331588745, -1.3890494108200073, -0.5689836144447327, -0.9692908525466919, 1.2876728773117065, -0.1729283332824707, 0.5984988808631897, 0.22022292017936707, -1.1772947311401367, -0.1788446009159088, 0.22251230478286743, -0.9240231513977051, 1.484939694404602, -0.9864821434020996, 0.9469887018203735, -0.15937559306621552, 0.301520437002182, 0.8692092895507812, -0.0018369900062680244, -0.98667311668396], [-1.8373770713806152, -0.2682502269744873, -0.8232230544090271, -0.9564899206161499, 1.2781199216842651, -0.0038733540568500757, -0.749669075012207, 0.2576538026332855, 1.1431975364685059, -0.5495903491973877, -1.9207773208618164, -2.4434173107147217, 1.043535828590393, 0.6737493276596069, -0.43118801712989807, 1.958932876586914, 0.9700409770011902, 0.001958867534995079, 0.17385026812553406, 1.713886022567749, -0.9942591786384583, 0.6307318806648254, 0.6894998550415039, 0.4834788739681244, -1.6471232175827026, -0.2762034833431244, 0.1873142421245575, 0.8556144833564758, -0.7311853766441345, -0.10121336579322815, 0.10900364071130753, 0.9470870494842529, -0.6330157518386841, -0.750882089138031, -0.2219981998205185, -0.5516341924667358, -1.050371527671814, -1.3701720237731934, -0.536250650882721, 1.4704782962799072], [0.05632395297288895, 0.7030909061431885, 0.7081834077835083, 1.0781238079071045, -0.4701743423938751, -0.19393514096736908, -1.1109684705734253, 0.6257840394973755, -0.1905531883239746, -1.0307207107543945, 0.024145061150193214, 0.10871496051549911, 1.801490306854248, -0.42781609296798706, -0.40661612153053284, 1.0904667377471924, 0.5967702865600586, 1.0398274660110474, -1.9008363485336304, 0.652291476726532, -1.3945889472961426, -0.8350645303726196, -2.227321147918701, -1.4767401218414307, -0.4532729387283325, 0.34112903475761414, 0.9131834506988525, -1.297268033027649, -1.3693565130233765, 1.48464834690094, -0.6917452812194824, 1.0333225727081299, -1.4032188653945923, 0.35005834698677063, 0.6011454463005066, -1.5602421760559082, -0.2673545479774475, 0.04930255562067032, 0.31501978635787964, -0.9701457619667053], [-0.37192729115486145, 0.2261652946472168, -1.0483415126800537, -0.5072342157363892, -1.5247722864151, -0.3158781826496124, 1.0035806894302368, 0.6321638226509094, 0.11175794154405594, -1.0477687120437622, 0.4604180157184601, -0.6599116921424866, -0.23648178577423096, 0.5653213858604431, 0.5736787915229797, -1.5846574306488037, -0.0847146213054657, 1.9176278114318848, -0.0757955014705658, -0.8060423731803894, -0.1558443307876587, 0.3660358786582947, 0.861527144908905, 0.8261350989341736, 1.2742279767990112, -0.0328892357647419, 1.0463273525238037, -0.9293543100357056, -0.809366762638092, -0.4961869418621063, -0.3763047456741333, -1.2548363208770752, 0.5655073523521423, 0.21072563529014587, 2.50424861907959, 0.23274177312850952, -0.25144773721694946, -0.18141266703605652, -1.2532756328582764, -0.39426907896995544], [1.5123752355575562, 0.878909170627594, -0.2556230127811432, 0.34345510601997375, -0.5759148001670837, -0.4158860445022583, 0.7702108025550842, 0.6378791332244873, 0.18233135342597961, -2.647413492202759, 0.10500016063451767, -0.04130018502473831, -0.10436603426933289, -0.4012798070907593, -0.5984877943992615, -0.5213597416877747, -0.36905211210250854, 0.6023039221763611, -0.5774087905883789, 0.32729023694992065, -0.9629297852516174, 1.5863101482391357, -0.4555247724056244, 0.6764222383499146, -1.827474594116211, 1.498022198677063, 0.02512410283088684, -0.09472433477640152, -0.6338450908660889, -0.4444955587387085, -1.2508479356765747, 1.4548206329345703, 1.0280401706695557, -1.2785643339157104, -1.2554446458816528, -0.012379568070173264, -0.20978480577468872, 0.4716051518917084, 0.3832460045814514, 1.3027406930923462], [-1.2268166542053223, -0.09146703034639359, -0.15542902052402496, 1.222680926322937, -0.43293818831443787, -1.2192891836166382, 1.3372591733932495, 0.9346259236335754, -1.5260521173477173, -0.6971396207809448, 0.7925884127616882, -0.343694269657135, -0.13299699127674103, 0.11333348602056503, 0.7513487339019775, 1.2142958641052246, 0.052697837352752686, 1.0374929904937744, 2.4938385486602783, 0.504149854183197, -0.03984207659959793, -0.9082984924316406, 0.36512482166290283, 0.6214942336082458, 0.2965352535247803, 0.10290233790874481, -0.44497427344322205, 0.4127563536167145, -0.550922155380249, 1.6917322874069214, 0.35727158188819885, 0.23477675020694733, 0.577850878238678, 0.6169754266738892, -0.3748459815979004, 1.406017541885376, -1.711792230606079, 0.0833190456032753, -1.0828595161437988, -0.23956410586833954], [0.7481544613838196, -0.5398176908493042, 0.20499901473522186, 0.8640822768211365, -0.7600826025009155, -1.1159206628799438, -0.19691935181617737, 1.574510097503662, -2.3184123039245605, -0.018117019906640053, 1.244136095046997, -1.2595081329345703, 0.28706762194633484, -0.36895987391471863, -1.2085692882537842, -0.9564278721809387, 0.2867245376110077, -1.6065235137939453, -2.1932547092437744, 0.045668065547943115, 0.896213948726654, 0.6256238222122192, 0.7244212031364441, 0.0779346153140068, 1.0805225372314453, -1.3678429126739502, -0.4126451909542084, -0.9366608262062073, -1.4266395568847656, 0.07739673554897308, -1.2186678647994995, -0.6454077959060669, -0.7388596534729004, -0.17711813747882843, -0.812275230884552, -1.6732953786849976, -1.9077999591827393, -0.6454506516456604, 0.10189229995012283, 1.5654574632644653], [-0.555121123790741, 1.3362526893615723, -0.9568761587142944, 0.45134320855140686, 0.6760417819023132, -0.144292950630188, 0.08443478494882584, -0.3151618242263794, 2.0733776092529297, -0.9058502316474915, -0.6739936470985413, -0.33973386883735657, 1.0737162828445435, -0.49870505928993225, 1.030100703239441, -2.343106508255005, -1.6773138046264648, -1.2061223983764648, 1.249330759048462, -0.7379253506660461, 0.9666579961776733, 0.03706836327910423, -1.491515874862671, -0.4621213674545288, -0.14149637520313263, 0.72590571641922, -0.006590200122445822, 0.2150173932313919, -1.2726730108261108, 1.2895950078964233, -0.6035467386245728, -0.3200783133506775, 0.5801234841346741, 0.9666316509246826, -0.48731529712677, 2.3641164302825928, 0.5195603370666504, -0.43074730038642883, -1.0840586423873901, 0.8261611461639404], [0.41058865189552307, 1.0126951932907104, 0.10591592639684677, -1.4698946475982666, 0.5474923253059387, 1.578976035118103, -0.43936797976493835, -0.43083885312080383, -2.5078492164611816, -1.1911410093307495, -1.4817473888397217, -1.1433987617492676, -1.0957831144332886, -0.5499396324157715, -0.6733723878860474, 0.8935526013374329, 0.1415911614894867, -0.6721594929695129, 1.063818335533142, -2.25246000289917, 0.932524561882019, -0.04090581461787224, -0.38765397667884827, -0.91754549741745, -2.549980878829956, -0.11341726779937744, 0.28169772028923035, 0.15514157712459564, -0.913764476776123, 0.23537695407867432, -0.06589269638061523, 0.26468324661254883, -0.5174822807312012, 1.278357744216919, 0.6832862496376038, 0.8831299543380737, -1.2123297452926636, -0.6388481855392456, -1.4104856252670288, -0.2101106196641922], [0.8691843152046204, -1.8737738132476807, -1.6032414436340332, -0.45308637619018555, 1.5328174829483032, 0.08606430143117905, -0.581618070602417, -1.4048556089401245, -0.20778292417526245, 0.5946053266525269, -1.8775345087051392, 1.1405760049819946, 2.214911937713623, 1.0714688301086426, 1.0271285772323608, 0.6696873307228088, 0.947391152381897, 2.535486936569214, 0.9007061719894409, -1.7485932111740112, -0.6083818078041077, -0.03855191171169281, -0.8236619234085083, -1.5488306283950806, -0.5946944952011108, 0.5959327220916748, 1.8292316198349, 0.08193020522594452, 0.8077284693717957, 0.5239366292953491, 0.5679069757461548, -1.04189932346344, 1.485816478729248, -0.17106501758098602, 1.0947706699371338, -0.0457988977432251, -0.015794189646840096, -0.31244710087776184, -1.127219796180725, 0.82318514585495], [2.212325096130371, 0.49749884009361267, 0.52897709608078, -0.25531983375549316, 0.8776435852050781, -1.9317702054977417, 0.9910902976989746, 0.3791321814060211, 0.6188366413116455, 0.3846428096294403, -0.6677908301353455, 0.1829231083393097, -2.3231334686279297, 1.6073122024536133, 0.20508414506912231, 1.0272806882858276, -0.8615541458129883, -0.11595837026834488, -1.113125205039978, -1.3911283016204834, 0.3828464448451996, -0.27853745222091675, 0.08480250090360641, -2.20609450340271, -0.7981690764427185, 0.5137048959732056, -1.1174505949020386, 0.884041428565979, -0.5848855376243591, 0.6137913465499878, -0.291300505399704, -1.264253854751587, -0.8086715936660767, -1.8140099048614502, -1.5203654766082764, 0.4120023548603058, 0.03073531948029995, -0.21526338160037994, -0.14179623126983643, 0.374362975358963], [-1.315398931503296, -0.1566297709941864, 1.4795641899108887, 1.1144155263900757, 0.08129309117794037, -0.7380160689353943, 1.1831953525543213, -1.0039010047912598, -0.9686116576194763, 1.171608567237854, 0.03295139968395233, -0.9489626288414001, 0.6212030649185181, -0.3554907441139221, 0.8002948760986328, -1.6296137571334839, -0.01570618525147438, -1.0279595851898193, 0.16129201650619507, 0.040264662355184555, -0.5863957405090332, 1.1422314643859863, -1.1118437051773071, 1.9064849615097046, -1.0744092464447021, 1.3250173330307007, 0.10123822838068008, 0.44677191972732544, 0.49802982807159424, -0.10635887831449509, 0.8073688745498657, 0.28983044624328613, 0.8900116682052612, 0.4364469647407532, 0.6960566639900208, -1.5332884788513184, -0.2242768555879593, 0.5223846435546875, 0.8970778584480286, 0.1572932004928589], [0.3577081263065338, -1.3095550537109375, 1.5270044803619385, 0.3130682706832886, 0.14503303170204163, -1.8576912879943848, -0.05880534276366234, -1.862706184387207, -0.8572568297386169, -0.24346864223480225, 0.30263614654541016, -0.07118118554353714, 0.5853014588356018, -0.7469404935836792, 0.7356187701225281, -0.493066668510437, -2.2480576038360596, 0.8749274611473083, -1.0295398235321045, 1.307023286819458, -1.599581241607666, 0.9337064027786255, -1.568292498588562, -2.399458646774292, 0.5740498900413513, -0.35217151045799255, -0.23587121069431305, 0.5119784474372864, 0.4827365577220917, -1.3742719888687134, -1.0739097595214844, -0.22742243111133575, 2.4020283222198486, -1.3450266122817993, -1.2854578495025635, 0.41369685530662537, -0.724374532699585, 0.9966674447059631, -0.12000050395727158, 1.1307916641235352], [0.3922923803329468, 0.1176050677895546, -0.08329601585865021, 1.8351300954818726, -0.8973138332366943, 1.1961913108825684, 0.6109546422958374, -0.6575181484222412, -0.4134863615036011, 1.1864032745361328, 0.1317245364189148, 0.4956967532634735, -1.4740469455718994, 0.2680310010910034, 0.3328299820423126, 2.106891632080078, -0.8626676201820374, -0.5283141136169434, -0.8061161041259766, 0.5363900065422058, 0.40274834632873535, 0.38455823063850403, 0.6690064668655396, 0.05151347070932388, -0.6291003227233887, 1.6425279378890991, -0.35106852650642395, -0.37073615193367004, 1.2437435388565063, 0.24574072659015656, -0.6827255487442017, -1.9882595539093018, -1.2795257568359375, -0.7442649006843567, -1.0421953201293945, -1.093930959701538, -0.9778167009353638, -0.6921375393867493, 0.5199100375175476, -0.17445921897888184]], "dtype": "float32"}, 3, 32], "e": {"__tup__": [{"__nd__": [[-2.4776647090911865, -2.2496914863586426, -4.856154441833496, 14.790842056274414, -7.179690361022949, 8.265798568725586, -15.014542579650879, 14.355247497558594, -14.22719955444336, -19.978893280029297, -14.55801010131836, -1.370255708694458, -0.17933064699172974, -20.167551040649414, 27.954282760620117, -15.64367961883545, 4.746612548828125, 0.19471225142478943, -20.665674209594727, 0.42477768659591675, -11.324481010437012, -7.7172160148620605, -11.546697616577148, -29.41091537475586, -6.98278284072876, 13.664323806762695, -12.736370086669922, 9.414127349853516, 7.170577049255371, 3.0960299968719482, -0.6673068404197693, -10.528335571289062, 43.765289306640625, 15.183065414428711, -3.1104679107666016, -2.108489513397217, -7.7774434089660645, -7.088935375213623, -4.8039937019348145, -10.37401008605957], [7.645638465881348, -0.39927002787590027, -13.055739402770996, 6.962993144989014, 7.607469081878662, -3.3215558528900146, -13.711006164550781, -12.242936134338379, 1.5231754779815674, 9.760051727294922, -20.17913246154785, -5.22302770614624, -33.184932708740234, -25.798124313354492, 22.12339210510254, -14.337695121765137, -19.2188720703125, -23.345632553100586, -28.072986602783203, 4.229809284210205, -30.96979331970215, -8.16372299194336, -14.22125244140625, -11.00969409942627, 13.55528736114502, 7.0909504890441895, -7.795563220977783, -24.797420501708984, -19.583890914916992, -7.152207851409912, 1.8449586629867554, 8.98885440826416, -0.3686150014400482, -10.815825462341309, 24.326059341430664, 23.818031311035156, -6.408464431762695, -3.6445159912109375, 7.129112720489502, -16.32874870300293], [-0.8967982530593872, -2.040273427963257, -1.198981761932373, 7.219399452209473, 0.5457102656364441, 6.072231769561768, 23.793729782104492, -31.62198829650879, 4.307916641235352, -20.523744583129883, 9.689567565917969, 1.1084800958633423, 40.91253662109375, 28.65742301940918, 8.403489112854004, 9.98302936553955, 14.71097469329834, 18.608287811279297, -3.584350824356079, 38.8590087890625, -20.351055145263672, 18.065271377563477, -11.245590209960938, -12.608280181884766, -19.460311889648438, 8.272189140319824, -4.680022239685059, 9.486892700195312, -11.681462287902832, 32.05867004394531, 0.6083391904830933, -5.434563636779785, 20.361164093017578, -11.763278007507324, -6.628880500793457, -15.001553535461426, -35.689720153808594, 3.3425614833831787, 10.704666137695312, 9.727892875671387], [-32.17795944213867, -20.112293243408203, 24.25164794921875, 5.187983512878418, 3.1109976768493652, -8.845512390136719, 32.5482063293457, 18.72549819946289, 3.368490219116211, 21.892391204833984, -4.768929481506348, 4.9295878410339355, 32.544551849365234, 7.7707343101501465, -24.545955657958984, -31.405555725097656, -1.1772924661636353, 3.2717881202697754, -15.443180084228516, 1.800094485282898, 22.325944900512695, 8.088332176208496, -17.76584815979004, -14.840084075927734, -3.7779781818389893, 13.414030075073242, 14.403839111328125, -8.937723159790039, 14.838545799255371, 24.938398361206055, -13.443796157836914, 6.559054374694824, -2.8829245567321777, 16.338603973388672, 30.000638961791992, -26.315980911254883, -21.746679306030273, -7.559895038604736, -15.934053421020508, 36.75421905517578], [10.578630447387695, -5.024216651916504, -0.8554676175117493, -1.9102210998535156, -2.3753533363342285, -10.387638092041016, 38.597686767578125, -23.357284545898438, 10.689449310302734, 6.930323600769043, 17.36188507080078, 10.725348472595215, -7.999009609222412, 20.5952205657959, 36.595619201660156, -9.827281951904297, 14.311454772949219, 0.383924275636673, 11.89737606048584, 15.235954284667969, 0.816670298576355, -22.180652618408203, 20.915929794311523, -11.852092742919922, -20.489351272583008, 4.221144199371338, 4.147409915924072, 17.697467803955078, 26.28140640258789, 0.40598270297050476, 12.491683959960938, 4.600523471832275, 4.380117416381836, -7.791090488433838, 32.10269546508789, -17.144060134887695, 10.749896049499512, -1.334175944328308, 46.138126373291016, 20.82845687866211], [31.670730590820312, 6.050449371337891, 48.056461334228516, -8.84831714630127, 15.248974800109863, 28.403249740600586, -16.022859573364258, 33.91862487792969, -23.415254592895508, -11.332426071166992, -0.3516744077205658, 1.4503333568572998, 30.7493839263916, -24.702302932739258, -0.08568794280290604, 29.905406951904297, 24.014495849609375, -21.187320709228516, -6.656007766723633, 11.680960655212402, -5.776979446411133, 13.176340103149414, 7.05409049987793, -35.1757926940918, 27.63558578491211, 9.393535614013672, 3.1007988452911377, 8.370314598083496, -13.574557304382324, -9.106337547302246, 24.589658737182617, 15.792322158813477, -14.494524955749512, -15.991790771484375, 20.143390655517578, 30.50062370300293, 0.9903265237808228, 5.552626132965088, -7.748180866241455, -29.44732093811035], [35.711761474609375, -9.270916938781738, 5.408203125, -29.8778018951416, -0.8113474249839783, 0.6072892546653748, 10.606221199035645, -20.629592895507812, -17.767642974853516, 8.964158058166504, -3.5930399894714355, 2.578667163848877, 5.793854713439941, 9.246407508850098, 15.039040565490723, 23.394676208496094, -12.022916793823242, -1.6047567129135132, -6.839844226837158, -34.17988967895508, -31.272253036499023, -5.992581367492676, 5.908714294433594, -1.0587282180786133, 21.058670043945312, -6.12194299697876, 1.5117050409317017, 15.517228126525879, 3.661093235015869, -5.919763088226318, -11.12060546875, -1.425474762916565, -14.003181457519531, 9.117376327514648, -16.201366424560547, 24.019657135009766, 15.473278045654297, -7.60139274597168, 16.157363891601562, -13.814298629760742], [-21.234451293945312, -28.901111602783203, 15.225341796875, 8.417417526245117, 36.225154876708984, -5.670123100280762, -13.96310806274414, -3.799654245376587, 10.85551929473877, -16.47328758239746, 11.342279434204102, 4.631471633911133, -10.341878890991211, 0.2942463755607605, -13.84538459777832, -32.95232009887695, -12.594013214111328, 20.624889373779297, 11.543142318725586, -29.119056701660156, 16.627138137817383, 15.367588996887207, 15.685220718383789, -3.945056676864624, -22.87119483947754, -4.068639755249023, 28.07994270324707, -4.476022720336914, -26.31097412109375, -7.664402961730957, -3.0417943000793457, -0.5340099334716797, -14.38700008392334, 5.310302257537842, 12.293773651123047, 24.402374267578125, 9.894301414489746, 13.176287651062012, 2.8616280555725098, -11.422697067260742], [-11.274773597717285, -35.2677116394043, 23.953176498413086, -0.9900030493736267, -2.197671413421631, -3.4325854778289795, 19.73712730407715, 3.0886316299438477, 5.08554744720459, 0.3836546540260315, -13.32372760772705, -32.64933395385742, 2.994887113571167, 3.8415865898132324, 30.133296966552734, 21.48773956298828, -5.696696758270264, 3.040834426879883, -18.40229034423828, -30.821386337280273, -11.03821086883545, 20.26038932800293, 3.557086944580078, 15.845829010009766, 7.163345813751221, 22.29379653930664, -20.27943992614746, -19.015583038330078, 10.523347854614258, -23.581987380981445, 28.64840316772461, 5.90640926361084, -10.560837745666504, -16.917938232421875, 10.959315299987793, 4.242502212524414, 6.49383020401001, -1.6163105964660645, 8.013664245605469, -0.8536630272865295], [-15.033287048339844, -3.7888705730438232, 13.643054962158203, -30.22038459777832, -23.446693420410156, -9.916404724121094, -20.965688705444336, 30.867204666137695, 24.233295440673828, -29.903350830078125, -21.216415405273438, -7.359382629394531, -22.41464614868164, -6.777231693267822, -14.377802848815918, 4.709107398986816, 8.151249885559082, -40.29316329956055, -17.380399703979492, -17.22644805908203, 3.841403007507324, 10.516683578491211, -0.13170485198497772, 3.7183353900909424, -12.611737251281738, 9.573405265808105, -14.847752571105957, 29.33946990966797, 0.35379183292388916, -6.040860176086426, -34.439266204833984, -9.45844841003418, 20.418346405029297, -40.752174377441406, 17.36895179748535, 31.83704376220703, 15.374727249145508, 39.30353546142578, 11.541635513305664, 4.310436248779297], [-6.752706050872803, -26.047359466552734, -16.81200408935547, -1.5948388576507568, -2.044205665588379, -20.405344009399414, 32.20319366455078, -5.040083408355713, 25.128753662109375, 0.39812222123146057, -23.7518367767334, 30.32401466369629, 21.35785484313965, -18.050228118896484, 21.36150360107422, -21.1159725189209, -8.269364356994629, -5.913010597229004, 30.608720779418945, 2.374762535095215, -8.195090293884277, -11.07127571105957, -30.40209197998047, -6.045483112335205, -24.145645141601562, 15.659587860107422, 0.7030225396156311, 25.222837448120117, 6.8564348220825195, 16.454578399658203, 5.398163795471191, 3.9579272270202637, 1.8624134063720703, -10.897500038146973, 20.374433517456055, 24.337207794189453, -1.799788236618042, 25.0340518951416, 12.024578094482422, -10.914647102355957], [-12.978128433227539, -1.5753387212753296, -16.529415130615234, -14.621211051940918, -4.598881721496582, 5.823461532592773, 14.623753547668457, 31.654279708862305, 13.236490249633789, 20.63312339782715, 0.8541197776794434, 0.22292907536029816, 25.443328857421875, 39.427978515625, -28.72673225402832, -13.019866943359375, 0.6508602499961853, -10.013874053955078, -5.397480487823486, 12.688199043273926, 32.12301254272461, -42.345726013183594, 23.5606689453125, 19.026866912841797, -26.06691551208496, -22.939226150512695, 1.067313313484192, -8.877211570739746, 10.302245140075684, 5.37797212600708, -16.557390213012695, -9.473876953125, -10.81423568725586, 7.497396469116211, 10.888325691223145, -28.88695526123047, 6.79191255569458, 0.5546690821647644, 8.327435493469238, 45.3335075378418], [21.574813842773438, 5.427680969238281, 18.090755462646484, -1.8298392295837402, -13.881060600280762, -17.65856170654297, -18.20647621154785, -15.442214965820312, 16.136093139648438, 9.310731887817383, -4.456244945526123, -2.3999929428100586, -17.436508178710938, 9.639199256896973, -3.694965362548828, 13.861557960510254, 36.15055847167969, -1.468567967414856, 14.587508201599121, 15.975151062011719, -2.479989767074585, -10.724773406982422, 5.697809219360352, 11.584712982177734, 12.462451934814453, 12.740203857421875, -7.796989917755127, 0.6596025824546814, 22.91655921936035, -33.653446197509766, 1.9609878063201904, -16.23468017578125, -7.7587504386901855, 1.696108341217041, 32.49579620361328, -11.383600234985352, -3.6958799362182617, 21.08441162109375, 23.571481704711914, -2.190680742263794], [-31.243066787719727, 1.740064263343811, 21.691434860229492, -2.6897151470184326, 19.861366271972656, -13.5714111328125, -24.502565383911133, 32.66715621948242, -10.16346263885498, -13.539621353149414, -1.9830454587936401, 1.6030020713806152, 19.55261993408203, 0.6873031258583069, -23.634178161621094, 34.657371520996094, 15.807043075561523, -8.840744972229004, -0.06575413793325424, -48.90840530395508, 33.414337158203125, 21.907455444335938, -1.4813833236694336, 36.694549560546875, 4.877845287322998, -15.34221363067627, -6.726480007171631, -4.458072662353516, -4.937670707702637, 13.14360237121582, -16.120174407958984, 28.85404396057129, -11.697047233581543, 10.194512367248535, 11.207191467285156, 10.979386329650879, 0.8990775346755981, 0.7148690819740295, -18.51695442199707, -1.585207462310791], [14.412720680236816, 13.42896556854248, 25.94447898864746, -23.026758193969727, 8.441753387451172, -11.54952621459961, 12.37460994720459, -11.388556480407715, 1.7078397274017334, 4.132486343383789, 17.162290573120117, -4.206655502319336, -18.849166870117188, -20.85176658630371, -41.166709899902344, -25.98434066772461, -7.483363151550293, 20.422277450561523, 18.272727966308594, -26.7182559967041, -7.640804767608643, 13.585919380187988, -8.872173309326172, -13.020357131958008, -6.411139965057373, 5.445173263549805, -15.199380874633789, -30.061887741088867, 15.670310020446777, -1.2543399333953857, 27.855859756469727, 21.862754821777344, 11.023097038269043, 10.848033905029297, 9.713263511657715, 21.299041748046875, 14.286816596984863, 28.417875289916992, -19.00632095336914, -14.001276969909668], [0.9235895276069641, -27.570281982421875, -0.7353830933570862, -10.865323066711426, -14.205442428588867, -7.027818202972412, 10.980283737182617, 1.0988327264785767, -9.92313289642334, 19.068828582763672, -10.669014930725098, 9.553718566894531, -9.079432487487793, -15.87949275970459, 31.583602905273438, 5.113143444061279, 14.803664207458496, 42.07587814331055, -15.672019004821777, -9.189837455749512, -2.901376724243164, -9.017212867736816, -13.335230827331543, 1.4776666164398193, -9.790413856506348, 22.84115982055664, 28.782133102416992, 9.002677917480469, -3.786384105682373, -5.865734100341797, 12.438089370727539, 35.87459945678711, -27.45069122314453, -12.596940040588379, 4.299055576324463, -4.560431480407715, -16.90878677368164, -15.912795066833496, -4.971371650695801, 19.20730209350586], [23.476581573486328, -10.047418594360352, -4.81963586807251, 10.281837463378906, -22.664297103881836, -3.3619751930236816, 19.53753662109375, -2.4097812175750732, -22.088611602783203, -3.18642520904541, 3.581413507461548, 6.273316383361816, 22.359661102294922, 36.293216705322266, 7.6260576248168945, -8.256525039672852, 17.934968948364258, 1.7006794214248657, -27.01169204711914, 16.442893981933594, 31.176658630371094, 8.478439331054688, 15.135517120361328, -7.547635555267334, -1.65900456905365, -13.410837173461914, 13.633559226989746, 13.53332805633545, 9.052000045776367, -19.585664749145508, 18.308837890625, 18.83940887451172, -3.869920492172241, 11.582185745239258, 11.340498924255371, -5.53582239151001, 10.963544845581055, 3.284165620803833, 1.7383191585540771, 23.634244918823242], [-14.837225914001465, -11.419584274291992, 6.191572189331055, 4.468090057373047, -3.330995559692383, 11.346658706665039, 24.074146270751953, -42.578590393066406, -6.467332363128662, 37.3342170715332, 10.372467994689941, -2.8641085624694824, -13.479086875915527, 9.670201301574707, 9.455842018127441, -7.567025661468506, 28.403383255004883, 4.75730037689209, 33.4784049987793, -16.350631713867188, 4.916423797607422, 19.567899703979492, -12.478306770324707, -10.585064888000488, 3.388587474822998, 13.814240455627441, 5.040280818939209, 19.941091537475586, 3.795818328857422, 16.39207649230957, 32.06019592285156, -21.675579071044922, -15.79862117767334, 10.244808197021484, -4.208188056945801, 19.615367889404297, 36.49957275390625, 19.675201416015625, 1.2441685199737549, -28.661901473999023], [23.716218948364258, -18.85546875, 8.05710506439209, -35.10622024536133, 21.414871215820312, -4.4974870681762695, 3.36611270904541, -12.760549545288086, 1.7862842082977295, -4.8057708740234375, 23.59095573425293, 26.72576904296875, 26.929094314575195, -1.796213984489441, -3.775183916091919, -10.539103507995605, 16.991579055786133, -4.733482360839844, -10.339666366577148, 29.401607513427734, -0.6425502896308899, -42.516998291015625, -13.07468032836914, -28.040498733520508, -17.992460250854492, 20.107322692871094, 14.056032180786133, -10.674311637878418, 8.964985847473145, 12.367117881774902, 10.240260124206543, 67.13374328613281, 15.779722213745117, -5.266931533813477, -15.515973091125488, 18.469228744506836, 1.6287307739257812, 6.488185405731201, -18.516881942749023, -9.345734596252441], [7.721925258636475, -3.264727830886841, -6.803825378417969, -15.188870429992676, 31.62242889404297, 43.04549789428711, -30.79365348815918, 16.443477630615234, -21.965776443481445, -17.530996322631836, 8.937926292419434, 18.53390121459961, -19.751249313354492, -2.323045492172241, -9.961042404174805, 19.950984954833984, 1.8229559659957886, -10.293787956237793, -3.8039112091064453, 17.45676040649414, -5.584835529327393, 10.050481796264648, -4.095715522766113, 15.076197624206543, 19.200754165649414, 17.677709579467773, 19.504383087158203, 1.4803458452224731, -8.017965316772461, 8.051098823547363, -13.852798461914062, 7.128710746765137, 2.5751333236694336, 16.251758575439453, -14.725096702575684, -5.9210100173950195, 7.742604732513428, 14.667403221130371, -4.427733898162842, -21.034374237060547], [17.374887466430664, -2.7496323585510254, -1.0621256828308105, 33.714168548583984, -14.167365074157715, 7.028862476348877, -1.393553376197815, -17.181072235107422, -17.287946701049805, -1.994373083114624, 8.526942253112793, 6.486478805541992, -21.476909637451172, -5.272507667541504, 1.2530407905578613, -6.681122303009033, 24.4811954498291, -11.038625717163086, -4.807018280029297, -2.6339900493621826, -26.21269416809082, -8.532198905944824, 14.367709159851074, -34.43307113647461, -15.94260025024414, -1.7380012273788452, -21.677860260009766, -5.80658483505249, 19.053667068481445, -6.992340564727783, 33.74309539794922, 11.533053398132324, 14.195300102233887, -0.014333146624267101, -18.251996994018555, -27.929960250854492, -11.002584457397461, -7.294055938720703, -20.10421371459961, -16.66861915588379], [29.90540885925293, 3.6123993396759033, -29.836957931518555, 6.328261852264404, 24.483861923217773, -8.336387634277344, 18.037395477294922, -20.936874389648438, 3.6373345851898193, -12.171697616577148, -13.904522895812988, -13.082181930541992, -4.707973957061768, 1.9063341617584229, 11.785623550415039, -25.874849319458008, 16.800426483154297, -8.350114822387695, 19.664958953857422, -4.333437919616699, 10.129196166992188, 1.8128353357315063, 5.6757588386535645, 5.196898937225342, -16.999826431274414, -0.5823681950569153, 9.078893661499023, -2.169156074523926, 2.1991095542907715, -1.1948518753051758, 43.73794174194336, -15.721435546875, 23.599109649658203, 9.524040222167969, 27.815067291259766, -3.3566901683807373, 19.28746795654297, 15.041606903076172, -12.708223342895508, 16.97543716430664], [7.394123554229736, -7.213632583618164, 18.229204177856445, -26.230043411254883, -11.416732788085938, 9.901955604553223, -13.64694595336914, -5.63345193862915, 11.842164039611816, -3.181156873703003, 2.3546955585479736, -5.963676929473877, -25.1030216217041, -8.387001037597656, -0.18103741109371185, -21.80706787109375, 5.956198215484619, 42.67958450317383, 1.918302297592163, -0.6289260983467102, -11.798583984375, 8.15621566772461, -18.391292572021484, 9.665872573852539, 13.726730346679688, 25.301822662353516, 0.6492276191711426, 15.306960105895996, 2.834846019744873, 15.015914916992188, 12.16159725189209, -36.08476257324219, -8.726716041564941, -7.6419219970703125, -13.457183837890625, -17.47083282470703, -0.2602441906929016, -4.581843376159668, 3.538954019546509, -2.4924697875976562], [14.497247695922852, -10.597678184509277, 3.167172908782959, 25.748807907104492, 16.933984756469727, -16.1484432220459, 11.916337013244629, -8.796747207641602, 13.811163902282715, 4.25304651260376, 18.233800888061523, -9.710283279418945, -14.032772064208984, 10.78265380859375, 10.796939849853516, 28.29208755493164, 19.092296600341797, -1.4898711442947388, 2.920295476913452, 0.46510040760040283, 25.052522659301758, -0.5875446200370789, 2.07108211517334, 28.428911209106445, -21.086530685424805, -3.771372079849243, 29.69240951538086, 23.750831604003906, -11.943242073059082, 17.848628997802734, 9.743895530700684, 20.820940017700195, -14.00411605834961, -11.603157997131348, 5.875251293182373, 28.317729949951172, 14.207967758178711, -25.910924911499023, 3.9582135677337646, 21.287063598632812], [25.683094024658203, 26.08138084411621, 30.698854446411133, -11.543463706970215, -24.337997436523438, -15.28834342956543, 16.237565994262695, -11.803317070007324, 5.381728649139404, -4.684175491333008, 14.032398223876953, 23.011064529418945, -24.075849533081055, 14.32033634185791, -10.15463638305664, -15.531067848205566, 44.44710922241211, -2.519212484359741, -10.429394721984863, -2.244347333908081, -24.21432876586914, -4.863720893859863, -12.9346923828125, -14.05840015411377, -11.39206314086914, -4.716151714324951, 1.6474334001541138, -13.915939331054688, -12.85434341430664, -5.918243885040283, 13.275012969970703, 13.996049880981445, -3.3777239322662354, -16.897172927856445, 15.734598159790039, 14.986915588378906, -5.636833667755127, -41.66754150390625, 26.452932357788086, -32.70712661743164], [-9.785377502441406, -3.7483668327331543, -3.6256675720214844, 8.428170204162598, 9.13861083984375, 18.01964569091797, -14.56554126739502, 26.090011596679688, 6.209778308868408, 12.256705284118652, -26.896251678466797, 14.414650917053223, 8.2291259765625, -6.9389190673828125, -15.765458106994629, 5.221614360809326, 25.202253341674805, -20.379962921142578, -17.64972686767578, 5.495147705078125, -17.37684440612793, -7.28009557723999, 4.040302276611328, -5.224053859710693, 52.04765701293945, -11.006372451782227, 38.810604095458984, -9.709280967712402, -2.4983818531036377, 27.446481704711914, -14.245372772216797, 1.195021152496338, -29.7880802154541, 11.004744529724121, 16.73130989074707, -10.762096405029297, 8.302271842956543, -5.021430492401123, 33.122039794921875, 3.647801399230957], [2.498265266418457, -4.835709571838379, 37.37411880493164, -31.21750831604004, 15.4541015625, 5.354487419128418, 4.976511478424072, -14.594368934631348, -0.3636879622936249, 11.988116264343262, -6.723880290985107, 14.686850547790527, -13.14404296875, 11.793532371520996, 1.0534290075302124, -2.9253017902374268, 6.038373947143555, -0.010736881755292416, -5.105031967163086, -2.6802000999450684, -2.5142576694488525, 14.279763221740723, 1.0357555150985718, 19.946123123168945, -7.743999004364014, 6.848687171936035, -22.801822662353516, -5.891915798187256, 20.96860694885254, 16.006559371948242, 11.574044227600098, -3.9413018226623535, -1.1831085681915283, -3.139565944671631, 4.79072904586792, 8.130415916442871, 2.36586856842041, 10.964862823486328, 16.716270446777344, 7.140022277832031], [9.378735542297363, 17.880937576293945, -24.742565155029297, -6.274491310119629, 7.819759845733643, 28.005403518676758, -2.4468345642089844, -20.351654052734375, 8.286748886108398, -36.86484146118164, -3.0532066822052, 1.5974501371383667, -3.0194480419158936, 13.726914405822754, 18.52480697631836, -13.672469139099121, 23.853466033935547, 20.80408477783203, 20.9016056060791, -10.39087963104248, -13.31820297241211, 17.862607955932617, 39.97030258178711, 1.4750925302505493, -21.53216552734375, 7.841094017028809, -13.489913940429688, -13.234527587890625, -24.409740447998047, -4.5244011878967285, -16.0516414642334, 22.952457427978516, 3.6155953407287598, 1.114364504814148, -17.267789840698242, -21.481300354003906, -0.7157924175262451, -16.351024627685547, -4.628929138183594, 2.859013080596924], [-33.37441635131836, -3.0006961822509766, 23.216848373413086, 6.75966215133667, -21.54110336303711, -4.099450588226318, 1.6942367553710938, 2.817335367202759, 8.418929100036621, 46.38808059692383, 41.55058670043945, -7.431637763977051, -18.746597290039062, 5.674009323120117, 3.5364882946014404, -11.719621658325195, -13.585665702819824, 41.62516784667969, 13.09942626953125, 21.819292068481445, -9.116538047790527, 2.7758560180664062, 10.942075729370117, -31.820383071899414, 3.9632277488708496, 6.1578264236450195, 2.602895975112915, 35.65140914916992, -4.411286354064941, -12.508042335510254, -16.1621150970459, 9.908880233764648, -19.87157440185547, 17.46422576904297, 2.692462682723999, 20.294553756713867, 28.443029403686523, -17.053421020507812, -17.127586364746094, -27.820613861083984], [21.036026000976562, 5.291898727416992, -13.499360084533691, 20.9630184173584, -9.464098930358887, 2.8116652965545654, -29.051164627075195, -38.44628143310547, 2.710634231567383, -14.773758888244629, 3.129514217376709, 28.757158279418945, 30.025768280029297, 26.87078094482422, -7.441845893859863, 11.670950889587402, 8.197932243347168, -2.9901623725891113, -18.0562686920166, -28.17823028564453, 3.093682050704956, -7.699421405792236, 21.322887420654297, -10.580623626708984, 10.349109649658203, 19.9814395904541, 31.424545288085938, -16.618425369262695, -6.5875115394592285, -10.41430950164795, -3.0492751598358154, 12.482919692993164, -0.7143442034721375, 14.636761665344238, 3.3629071712493896, -18.50177001953125, -20.989459991455078, 17.465532302856445, 26.478641510009766, -25.956960678100586], [12.00448226928711, -30.48967170715332, -28.934532165527344, 34.35852813720703, 29.95505142211914, 50.266719818115234, 21.643062591552734, -12.457999229431152, -26.394800186157227, -2.8700170516967773, 14.629589080810547, 5.175701141357422, 14.776999473571777, 1.217498540878296, 7.056338787078857, -26.54419708251953, 14.608132362365723, -10.306573867797852, -9.174391746520996, -29.15488052368164, 13.229466438293457, -4.385031700134277, 4.30714750289917, 16.081151962280273, 3.5978825092315674, -4.030938625335693, -14.751554489135742, -10.453213691711426, -20.353086471557617, -7.315711498260498, 17.0074405670166, -21.863250732421875, 10.130084991455078, 24.71122169494629, -17.649105072021484, -12.455525398254395, 19.682680130004883, -1.591921329498291, -3.6960787773132324, 14.929872512817383], [-10.631266593933105, 21.711639404296875, -13.254128456115723, -14.090536117553711, -33.45545196533203, -5.320156097412109, -19.10757064819336, -23.494430541992188, -24.67363166809082, 27.67349624633789, 5.414459705352783, 1.4738792181015015, 8.987074851989746, -7.523286819458008, 10.788355827331543, 22.803123474121094, 21.563438415527344, -6.864660263061523, 3.2941243648529053, 19.3494930267334, 7.319066524505615, 14.63925552368164, 0.933907151222229, 17.300739288330078, 0.9838435053825378, -4.577916622161865, 0.5970803499221802, -0.1945463865995407, -4.109686851501465, 2.991095781326294, 2.6309385299682617, -0.5368103384971619, 6.113758563995361, 13.01755142211914, 24.970579147338867, -1.3652129173278809, -0.7944725751876831, -27.20013999938965, 9.359782218933105, -0.011820964515209198], [-26.067752838134766, -28.401155471801758, 11.27182388305664, 9.569697380065918, -6.0763258934021, -11.137364387512207, -4.098647117614746, 0.48456278443336487, -1.4175668954849243, -7.26885986328125, -15.584381103515625, 3.9696264266967773, -9.91436767578125, 34.77421951293945, -7.040496349334717, -9.933022499084473, 19.649526596069336, 1.0592657327651978, 13.645895957946777, -16.25603675842285, 13.59941577911377, 10.192760467529297, 10.782394409179688, -22.08462905883789, 13.816967010498047, 12.295647621154785, -33.94811248779297, 5.82130765914917, 24.158857345581055, 11.191001892089844, 12.572271347045898, 1.5821442604064941, -6.432499885559082, 23.435441970825195, -10.341527938842773, 8.591889381408691, -16.000505447387695, 29.782459259033203, 5.2738494873046875, -22.119497299194336]], "dtype": "float32"}, {"__nd__": [[[-5.290738105773926, -7.6061530113220215, -1.4072504043579102, 6.989364147186279, -7.124671936035156, 2.4306752681732178, -15.373483657836914, -2.9703280925750732, -13.91891860961914, -12.739324569702148, -18.64395523071289, 5.851354122161865, 1.4069933891296387, -25.71857261657715, 10.745250701904297, 1.4186290502548218, -1.4272006750106812, -8.3330717086792, -13.836124420166016, 5.836709499359131, -9.997252464294434, 9.978548049926758, -13.512619018554688, -6.31615686416626, -15.795533180236816, 9.046504020690918, -2.8495564460754395, 4.613688945770264, -2.5402824878692627, -1.5811805725097656, -16.133520126342773, 1.991226315498352, 29.296037673950195, 4.997025966644287, 3.121903657913208, -0.9109053015708923, -6.770901679992676, 9.74317455291748, -20.31011390686035, 2.7324860095977783], [-1.0796983242034912, -4.410876750946045, 13.241205215454102, 8.355319023132324, 10.10659122467041, -8.982828140258789, -1.0494470596313477, -3.884450674057007, 8.756895065307617, 13.026496887207031, -13.606626510620117, 0.09730029851198196, -11.07287311553955, -17.95201873779297, 20.98296356201172, -6.622519016265869, -31.377676010131836, -6.281567096710205, -22.956581115722656, -15.167511940002441, -14.54272174835205, 3.4650518894195557, -14.49817943572998, -19.533491134643555, -3.148379325866699, 7.690528869628906, 19.495800018310547, -23.333820343017578, -17.89702606201172, -4.284543514251709, -8.500602722167969, -0.04264262318611145, 2.0406227111816406, -7.825417995452881, 27.186485290527344, 29.80717658996582, -3.175276517868042, -6.308076858520508, 12.088388442993164, -9.972991943359375], [5.851196765899658, 5.1040940284729, 7.866973876953125, -10.195292472839355, -12.983604431152344, 13.011579513549805, 2.8425545692443848, -5.001798152923584, -3.1569924354553223, -4.738726615905762, 6.465303421020508, -1.8381611108779907, 16.48366928100586, 2.874678611755371, -2.435748338699341, 12.046473503112793, 8.281173706054688, 18.289852142333984, 9.649083137512207, 26.61211585998535, -2.7670371532440186, 29.78106117248535, -12.89120864868164, -9.432035446166992, 3.5802879333496094, 18.42536163330078, -6.415000915527344, 4.948525428771973, -10.737646102905273, 13.735196113586426, -1.870400071144104, 7.670957565307617, 1.323364019393921, -10.799702644348145, -7.6759748458862305, 0.8629627823829651, -28.539731979370117, 6.145944118499756, 6.605429172515869, -21.3731746673584], [-17.471439361572266, -0.6001361608505249, 16.400585174560547, 6.531678676605225, 1.6342343091964722, -4.235534191131592, 17.612520217895508, 10.050899505615234, 6.403622627258301, 14.63420295715332, -16.12213706970215, -0.15030954778194427, 21.852252960205078, 10.419822692871094, -0.9044233560562134, -15.51878547668457, -12.966676712036133, 1.0539385080337524, 0.919586181640625, 5.864439487457275, 0.06713378429412842, -2.2689905166625977, -14.25389289855957, -14.904974937438965, 0.3707960247993469, 9.803906440734863, -11.799920082092285, -17.708690643310547, 3.0128955841064453, 8.29104995727539, 12.501662254333496, 1.4167842864990234, 2.1456210613250732, 22.935287475585938, 10.302610397338867, -8.428980827331543, -21.867097854614258, -9.70416259765625, 0.02257322520017624, 23.59176254272461], [8.00985336303711, -7.952675819396973, 6.730189323425293, -11.200033187866211, 0.23865151405334473, -9.265121459960938, 18.366588592529297, -21.89071273803711, -0.733083963394165, -3.001728057861328, 2.548279047012329, 4.566565036773682, 2.5815255641937256, -0.5810585618019104, 17.071210861206055, -9.321276664733887, 13.835236549377441, 3.994833469390869, 1.80915105342865, 10.090285301208496, -1.3875055313110352, 8.918853759765625, 10.32064151763916, -2.7929515838623047, 3.5403048992156982, 1.4965945482254028, 6.393822193145752, 15.184617042541504, 7.692578315734863, -13.981788635253906, -1.199710726737976, -5.543869972229004, 9.666719436645508, -1.0401291847229004, -7.762505054473877, -6.165770053863525, 13.280168533325195, -3.0245094299316406, 21.156047821044922, 7.003889083862305], [7.888939380645752, -2.9011447429656982, 26.07553482055664, 7.367095470428467, 2.231904983520508, 7.9488091468811035, -7.155716419219971, 3.3342206478118896, -12.620928764343262, -1.8392415046691895, -8.779836654663086, 1.7789201736450195, 18.802438735961914, -12.550627708435059, 19.07841682434082, 17.109390258789062, 9.89742374420166, 3.3091089725494385, -2.8042778968811035, 4.057909965515137, 2.4488940238952637, 8.745643615722656, 18.649635314941406, -27.2785587310791, 24.21255874633789, 3.1348397731781006, 0.8398261666297913, 9.67581844329834, -0.9510126113891602, -11.807811737060547, 17.239595413208008, -11.110261917114258, -3.3706068992614746, 1.0143204927444458, 13.584399223327637, 21.10115623474121, -10.119988441467285, -19.6788272857666, 5.0820088386535645, -8.116609573364258], [13.950812339782715, -2.1882920265197754, -11.16307544708252, -24.274200439453125, 5.780830383300781, 4.999995231628418, -5.270253658294678, -3.6844980716705322, -4.627224922180176, -0.4122680723667145, 3.829761266708374, 2.5626070499420166, 0.7087619304656982, 3.6828324794769287, 9.598416328430176, 5.82856559753418, -6.619687557220459, -11.688916206359863, -3.58190655708313, -4.729237079620361, -22.70288848876953, -4.589491844177246, 12.359222412109375, 5.766996383666992, 20.438251495361328, -7.499133586883545, 1.42276132106781, 8.784339904785156, 9.98885726928711, -8.142542839050293, -14.554667472839355, -0.6366049647331238, -8.191732406616211, -8.880271911621094, -6.730157375335693, 15.34158706665039, 29.87700653076172, -7.495698928833008, 7.55492639541626, -2.858628511428833], [-4.332622051239014, -7.636204242706299, -12.068907737731934, 10.499403953552246, 7.910526752471924, -7.794413089752197, -12.482131004333496, 3.1316137313842773, 5.991578102111816, -6.568141937255859, 5.062463283538818, 11.439729690551758, -7.925457000732422, 3.9835824966430664, -12.18663501739502, -19.79348373413086, -3.5496463775634766, 19.798660278320312, 11.383583068847656, -13.909873962402344, -10.129035949707031, 3.7762672901153564, 15.20572566986084, -7.4521684646606445, -12.502676010131836, 3.3995888233184814, 16.380146026611328, 0.6950931549072266, -2.200317621231079, 1.760323405265808, -0.42868471145629883, 1.780229926109314, -12.690598487854004, 0.7645288705825806, 1.3825345039367676, -9.884528160095215, 19.733776092529297, -3.9303030967712402, 0.029577486217021942, -4.523190975189209], [6.599417209625244, -14.876788139343262, 15.975733757019043, -0.3686366677284241, -2.637937068939209, -0.5011231303215027, 5.605304718017578, -1.3879797458648682, 21.793621063232422, 4.73928165435791, -13.458908081054688, -15.153337478637695, 12.53510856628418, -8.080028533935547, 13.467159271240234, -0.7091042399406433, -5.449461460113525, 1.7323819398880005, -9.761542320251465, -15.51341724395752, -10.930377006530762, 19.74408531188965, -3.2953200340270996, 12.674156188964844, -1.533773422241211, 19.81678581237793, -0.8815121650695801, -9.641168594360352, 13.652969360351562, -15.951581954956055, 1.2901792526245117, 9.905219078063965, 0.1364160180091858, -6.402075290679932, 15.350360870361328, 10.980364799499512, 6.031094074249268, 0.9916518926620483, 5.77241325378418, -5.963405132293701], [-7.567977428436279, 0.13887785375118256, 2.831857919692993, 4.6120710372924805, 9.396134376525879, -15.534991264343262, -13.432522773742676, 28.048484802246094, 10.221705436706543, -19.960437774658203, -15.179166793823242, -15.530120849609375, -12.52904224395752, -2.1249353885650635, -14.836705207824707, 5.617132186889648, 18.006092071533203, -7.377347946166992, -6.856735706329346, -15.887419700622559, -1.4213972091674805, 7.5846662521362305, -1.51466703414917, 9.096095085144043, -1.6003142595291138, 1.5166834592819214, -1.3157747983932495, 8.611512184143066, 2.7895991802215576, 7.607789516448975, -17.182674407958984, -12.510257720947266, 14.945043563842773, -7.327988624572754, 9.54869556427002, -3.5692524909973145, -7.222078800201416, 13.441767692565918, -4.184763431549072, -3.702474594116211], [-0.004724016413092613, -18.366363525390625, -18.35368537902832, -9.276105880737305, 12.693954467773438, -4.912848472595215, 0.3793703317642212, 17.335729598999023, 18.48200225830078, -5.086578369140625, 9.522490501403809, 11.754366874694824, 0.6874963641166687, -18.696102142333984, 4.97049617767334, -7.877455711364746, -7.260775566101074, 2.401059627532959, 12.461462020874023, 6.244009494781494, -10.46768569946289, -6.324373722076416, -9.79764461517334, -3.439758777618408, -8.4419527053833, 2.231971502304077, 12.559662818908691, -1.6508688926696777, 7.217321395874023, 8.313692092895508, 13.37451457977295, 15.324808120727539, -4.193397045135498, 8.276254653930664, 10.39899730682373, 6.19625997543335, 8.84676456451416, 6.1077880859375, 11.631735801696777, 6.728146553039551], [-12.835991859436035, -1.4159770011901855, -15.716913223266602, -14.633657455444336, -7.8796796798706055, 3.810199022293091, 12.562790870666504, 12.588510513305664, 6.897582054138184, 21.23928451538086, 7.957520008087158, 4.679346561431885, 0.964905321598053, 19.4757022857666, -31.94379234313965, -3.5971572399139404, -16.485498428344727, -14.424980163574219, 0.02404947765171528, -0.005838995799422264, 20.778881072998047, -27.08470344543457, -5.476632595062256, 9.088650703430176, -13.955598831176758, -8.516279220581055, -11.231454849243164, -4.3987932205200195, 0.6257200241088867, 5.886335372924805, 10.038381576538086, -9.01484489440918, -0.7383478879928589, 6.3105788230896, 10.831815719604492, -11.472003936767578, 1.0983870029449463, 3.557539701461792, -15.472293853759766, 18.948482513427734], [14.319951057434082, 6.94337797164917, 9.863158226013184, -3.2578701972961426, -7.194745063781738, -17.808338165283203, 8.932394981384277, 0.7587949633598328, 0.19214054942131042, -5.053018093109131, 0.6354470252990723, -8.443024635314941, -5.108791351318359, 1.0847891569137573, 11.1493501663208, 19.04609489440918, 10.9657621383667, -9.260501861572266, -3.3570897579193115, 16.776180267333984, -5.286250591278076, -6.767486572265625, 3.7110583782196045, 1.9656343460083008, 5.921881675720215, -2.1266419887542725, -16.668922424316406, 4.525559425354004, 9.578659057617188, -11.137613296508789, -4.11208963394165, -14.931124687194824, -1.0208606719970703, -6.341585159301758, 10.17750358581543, -9.092256546020508, -6.9499711990356445, 2.861156702041626, 12.842516899108887, -4.971297740936279], [-23.826412200927734, 5.642196178436279, 3.596701145172119, -6.458139419555664, 9.262237548828125, -12.814764022827148, -1.5521599054336548, 6.943305015563965, -11.888313293457031, -5.822799205780029, 1.5237574577331543, 4.449941158294678, 0.3257339894771576, 10.809541702270508, 2.671919107437134, 11.47085952758789, 17.11424446105957, 1.5903284549713135, 1.6129220724105835, -28.429353713989258, 10.809736251831055, 3.0622365474700928, -6.1790947914123535, 20.8858585357666, -5.102430820465088, -0.9434517621994019, -2.4230122566223145, -0.6497066617012024, 27.334911346435547, 10.232047080993652, -4.717414855957031, 8.277151107788086, -5.571163654327393, -3.3193752765655518, 20.859312057495117, -6.267369270324707, 0.6644484996795654, 21.698970794677734, -14.183327674865723, 6.759786605834961], [8.599356651306152, 7.447408199310303, 11.405858039855957, 1.589133381843567, 17.007780075073242, 1.9351216554641724, 14.722953796386719, 15.064224243164062, -2.75567626953125, -5.575312614440918, 12.493261337280273, -10.060389518737793, -4.740911483764648, -3.742717742919922, -14.944061279296875, -23.063617706298828, 6.281857490539551, 19.094924926757812, 13.227255821228027, -5.218204021453857, 3.137052536010742, 1.0768836736679077, -8.392603874206543, -0.9040846824645996, -14.159075736999512, -0.7411516904830933, -17.26001739501953, -10.444823265075684, 21.927637100219727, -4.089831829071045, 27.141035079956055, 9.747538566589355, -4.961337089538574, 8.400964736938477, 1.2866648435592651, 17.432861328125, 0.8774046301841736, 11.362506866455078, -6.706640243530273, 0.5226975679397583], [9.067633628845215, -26.042560577392578, 5.6647772789001465, -0.2626749873161316, -7.3671488761901855, -3.5972025394439697, 5.580417156219482, -14.911324501037598, 1.1306792497634888, 10.784451484680176, -4.796295642852783, 13.553832054138184, -0.07358431071043015, -14.82151985168457, 0.3500836193561554, 4.065467834472656, 5.075103282928467, 14.094733238220215, -20.324081420898438, 8.149957656860352, -5.935636043548584, -3.919297218322754, -4.062589168548584, 2.623868703842163, 2.5722196102142334, 1.6014059782028198, 18.630449295043945, 1.9740782976150513, -0.5131096243858337, 6.207225322723389, 11.459081649780273, 12.514531135559082, 6.299457550048828, -6.038383960723877, 3.5569324493408203, 13.687573432922363, 4.300579071044922, 3.097118377685547, 3.530832290649414, -4.167159557342529], [15.787875175476074, -9.17532730102539, 6.426949977874756, -6.963932037353516, -4.713349342346191, -11.299936294555664, 12.304893493652344, -2.5219333171844482, 0.21791963279247284, -0.02713673934340477, -4.120057106018066, -11.164138793945312, 6.547886371612549, 20.049882888793945, -0.14782676100730896, 8.93055248260498, 1.74574875831604, -8.821405410766602, -25.898344039916992, 9.80114459991455, 5.130192279815674, -5.713176727294922, 0.8299293518066406, -8.98463249206543, 0.18203367292881012, -8.006613731384277, 12.686193466186523, 19.485389709472656, 5.873723030090332, -2.235745906829834, 1.2282233238220215, 10.458306312561035, -3.983168840408325, 7.376636505126953, 5.742180824279785, 3.0189013481140137, 6.45046329498291, 8.9244966506958, 13.176104545593262, 2.237361192703247], [-7.282649040222168, -13.246438026428223, 5.654506683349609, 9.804033279418945, -4.3683671951293945, 3.2855632305145264, 11.394039154052734, -22.62978172302246, 0.44951343536376953, 27.38555335998535, 9.285451889038086, 2.4751267433166504, -7.541548728942871, -2.669372320175171, 12.041707038879395, -8.513456344604492, -3.951444625854492, 0.1879102736711502, 16.10474967956543, 20.146526336669922, 3.341188907623291, 2.0479326248168945, -3.5841028690338135, -4.033900260925293, 14.272873878479004, 4.638338565826416, 10.839170455932617, 4.355882167816162, -2.6254537105560303, 0.2310507893562317, 15.558816909790039, -6.474668025970459, 8.101365089416504, 7.194007396697998, -1.92145574092865, 8.865407943725586, 23.70068359375, 7.348100662231445, 11.023975372314453, -8.573974609375], [13.197519302368164, -18.60283088684082, 3.049079656600952, -15.105901718139648, -8.65717887878418, -2.7134902477264404, 5.536608695983887, -13.660042762756348, 4.117656230926514, -13.401079177856445, 1.4372332096099854, 11.13826847076416, 6.842440605163574, 0.04563454911112785, 12.505821228027344, -8.213844299316406, 0.5209559202194214, 15.152360916137695, 0.43716153502464294, 8.757819175720215, -5.064303398132324, -7.345158576965332, -4.631077766418457, -17.272293090820312, -0.8071910738945007, 15.6602144241333, -4.647663116455078, -10.73061752319336, 8.415295600891113, 3.263995409011841, 11.160099029541016, 33.85701370239258, 2.2704036235809326, -5.058664321899414, -23.112293243408203, 2.6548686027526855, 4.700562477111816, -1.23659086227417, -12.874114990234375, -13.246066093444824], [4.197896480560303, -7.229861736297607, 12.197026252746582, -0.17519083619117737, 5.082761764526367, 13.56052303314209, -18.74179458618164, 6.87562894821167, -17.959293365478516, -6.0799641609191895, 7.401181697845459, -0.44452762603759766, -6.558468818664551, -6.442411422729492, 0.9242281317710876, 18.54551887512207, 11.358630180358887, 4.8286895751953125, -15.336798667907715, 20.109445571899414, 15.856061935424805, 11.996387481689453, 0.8179944157600403, 14.562036514282227, 6.30654764175415, -6.273192405700684, 12.422565460205078, -3.669617176055908, -5.715998649597168, 19.596603393554688, -13.12112808227539, 9.998432159423828, 3.1084868907928467, 0.4503856897354126, -2.2832300662994385, 5.922318458557129, 1.0697566270828247, 5.5049824714660645, -8.178976058959961, -14.2539644241333], [0.38695043325424194, -9.210917472839355, 2.6514594554901123, 17.45990562438965, 3.0383617877960205, 9.622725486755371, -13.916289329528809, -3.3955230712890625, -3.1350395679473877, -5.488582611083984, 15.088907241821289, -8.090892791748047, -13.169793128967285, -13.357833862304688, 0.5878755450248718, -26.06357192993164, 8.752936363220215, -8.316791534423828, 8.716562271118164, -0.5104255080223083, -3.040203332901001, 7.869727611541748, -9.063344955444336, -3.845975637435913, -3.262058973312378, -15.886043548583984, -0.3997836709022522, 4.5078125, 12.298646926879883, -14.217763900756836, 16.099048614501953, 9.536540031433105, 15.645681381225586, 7.86263370513916, -13.016070365905762, -17.288249969482422, -0.7215239405632019, 3.5249788761138916, -4.657998085021973, 2.197408676147461], [13.334272384643555, 2.259556293487549, -13.713001251220703, -6.553020477294922, -5.090091705322266, -16.582698822021484, 14.27248764038086, -8.579874038696289, 1.9813040494918823, -6.679287433624268, 1.5248357057571411, 4.353699684143066, 1.852573037147522, -7.446398735046387, 1.1173083782196045, -19.112119674682617, -6.334677696228027, 5.104763507843018, 15.59316349029541, -10.418766021728516, 2.857379198074341, 6.11670446395874, 3.2620394229888916, -18.07593536376953, -22.248981475830078, 0.6715560555458069, 1.8228951692581177, -4.480297565460205, 1.6293162107467651, -4.856630802154541, 32.53337478637695, -12.867842674255371, 19.899316787719727, 15.66273307800293, 23.416683197021484, -4.869248390197754, 2.286740779876709, 13.098962783813477, -17.12479591369629, 2.8101258277893066], [-0.6522588729858398, 2.718317985534668, 21.148529052734375, -4.55921745300293, -5.485811233520508, 24.1130428314209, -6.4210309982299805, 0.6255636215209961, 11.320813179016113, -5.158886432647705, 7.808425426483154, -7.288744926452637, -3.897345781326294, 3.5418944358825684, -5.07226037979126, -0.5765928030014038, -9.521985054016113, 24.76428985595703, 1.5413517951965332, -1.0269533395767212, -5.372942924499512, -0.5495136976242065, -2.938581943511963, 0.8289615511894226, 6.160949230194092, 2.581409454345703, 9.81867790222168, 5.1098175048828125, -6.842041492462158, -3.110017776489258, -0.04884231835603714, -12.18240737915039, 2.687572717666626, -4.189733028411865, 4.923307418823242, 2.405015468597412, 1.7420244216918945, -4.43411111831665, 9.536105155944824, -9.930317878723145], [17.13758087158203, -9.617338180541992, -3.7535862922668457, 1.7421132326126099, 9.169804573059082, -3.028944253921509, 12.886017799377441, -5.493995666503906, 8.803335189819336, 1.8283385038375854, 19.028648376464844, 10.575613021850586, -15.406291961669922, 1.111242651939392, 10.945769309997559, 4.311592102050781, 10.52889633178711, 14.655291557312012, 8.311761856079102, 19.518749237060547, 9.276811599731445, 6.333378791809082, 7.054644584655762, 14.08062744140625, -13.04805850982666, -10.679251670837402, 22.172428131103516, 20.218244552612305, 6.316171169281006, -7.345037937164307, 3.7197587490081787, 16.300098419189453, 6.687609672546387, -2.770217180252075, -4.477950572967529, 10.479199409484863, 12.427370071411133, -13.984395027160645, 11.879121780395508, 15.40264892578125], [9.307372093200684, 6.168967247009277, 16.566286087036133, -14.1403169631958, -7.494492053985596, 1.353462815284729, 1.7694059610366821, -17.06546401977539, -4.617835998535156, -19.5142765045166, 3.044651746749878, 4.625289440155029, -19.78032684326172, 1.3382810354232788, -11.751002311706543, -15.116254806518555, 23.724889755249023, 3.4474174976348877, -11.79629135131836, 12.0964994430542, -21.227331161499023, 7.409377098083496, 9.402515411376953, -4.181707382202148, -10.112046241760254, -15.216588973999023, -3.554914951324463, -8.766361236572266, 9.604028701782227, -9.469112396240234, 1.268629789352417, 15.266827583312988, 18.825637817382812, -23.063594818115234, 0.8753446936607361, 7.709842205047607, -0.8814882636070251, -23.520122528076172, 32.972408294677734, -1.9524587392807007], [-5.137477874755859, 2.7507288455963135, 7.215036392211914, 4.431577205657959, 7.340606689453125, 9.71347427368164, -1.500512719154358, 13.938334465026855, 9.840020179748535, 2.651630163192749, 1.0282341241836548, 5.618338108062744, -12.029522895812988, -5.33428955078125, -13.096758842468262, 17.05230712890625, 16.54548454284668, -10.99236011505127, -7.213653564453125, 7.063035011291504, -15.763045310974121, -7.227776527404785, 18.893550872802734, -0.25008073449134827, 20.889394760131836, -5.257221698760986, 29.757749557495117, 13.455808639526367, -2.8305535316467285, 8.737502098083496, -12.684386253356934, -15.954416275024414, -24.273706436157227, 0.1654072403907776, -4.0806450843811035, 15.913723945617676, 0.04986076056957245, -23.264942169189453, 11.44044017791748, -2.8973424434661865], [13.955721855163574, -14.110382080078125, 4.0173749923706055, -9.79460620880127, 4.4109601974487305, 5.979278564453125, 8.3579740524292, -9.508283615112305, -0.2250477522611618, 12.44475269317627, -3.718078851699829, -5.571186542510986, -12.652621269226074, 6.731050491333008, -17.25266456604004, -11.810653686523438, 13.581330299377441, 4.8280487060546875, -13.530034065246582, -7.329412460327148, 1.9432017803192139, 10.82009220123291, 2.241614580154419, 2.7648496627807617, -11.343215942382812, -0.1859508901834488, -15.311243057250977, 4.396102428436279, 12.385668754577637, -1.6039661169052124, -0.49793779850006104, -6.158522129058838, 8.196958541870117, -8.356035232543945, -6.36505126953125, 1.3750849962234497, 5.531201362609863, 14.072511672973633, 2.6573948860168457, 7.763171672821045], [-7.443519592285156, 9.618193626403809, -8.097414016723633, -9.98786735534668, 4.634562015533447, 17.219675064086914, -6.59122896194458, -12.087541580200195, 12.351597785949707, -35.46215057373047, 5.876158714294434, 10.642614364624023, 1.5379284620285034, -6.9400153160095215, 5.945213317871094, -14.83540153503418, 11.553380966186523, 17.321155548095703, 15.791839599609375, -0.547670304775238, -13.18722152709961, 12.931438446044922, 19.782711029052734, -6.754776954650879, -10.550848960876465, -14.676314353942871, 7.3191046714782715, -7.616181373596191, -7.505725860595703, -5.3229169845581055, -6.610293388366699, 13.022432327270508, 1.1881108283996582, -13.096562385559082, -9.762866973876953, -7.6353583335876465, -6.937169075012207, -4.84340763092041, 0.2745402753353119, -4.48136043548584], [-11.429261207580566, 9.147271156311035, 20.02104377746582, 6.90360164642334, -8.506919860839844, -8.282167434692383, 2.4306607246398926, 6.800761699676514, -13.791903495788574, 14.886857032775879, 18.79192352294922, -9.36215877532959, -18.460052490234375, 9.675060272216797, 5.489336967468262, -11.82089614868164, -14.116692543029785, -2.26379656791687, 15.891963958740234, 4.023021221160889, -10.350080490112305, 3.4944517612457275, -4.530638217926025, -7.617281436920166, 11.817275047302246, -1.3633720874786377, -1.1384191513061523, 5.579169273376465, -9.437846183776855, 8.349486351013184, -1.0675559043884277, -3.6463353633880615, -8.961311340332031, 16.079648971557617, -18.2480411529541, 16.171493530273438, 27.298763275146484, -16.611379623413086, -10.871003150939941, -1.6555638313293457], [18.96820068359375, 3.256269693374634, -3.2767887115478516, -10.736473083496094, -5.867491245269775, -3.5088858604431152, -6.568068504333496, -12.641274452209473, 13.415966987609863, -13.638870239257812, 0.360084593296051, 7.044198989868164, 20.063411712646484, 16.270606994628906, -2.542759418487549, 12.072897911071777, 0.8620481491088867, -17.191020965576172, -3.467332363128662, -2.1522929668426514, 6.062162399291992, -14.544358253479004, 14.082893371582031, -0.5183643698692322, 8.534626960754395, 0.6505105495452881, 2.259925603866577, -15.407444953918457, 11.95645523071289, -5.17618465423584, 8.6259183883667, -7.973726749420166, 6.765608310699463, -0.8758351802825928, 13.237666130065918, -3.9034228324890137, -10.671283721923828, 24.993934631347656, 17.95598793029785, -21.962806701660156], [9.382162094116211, 0.4930248260498047, -22.336292266845703, 13.141153335571289, 8.39407730102539, 14.474103927612305, 4.150424957275391, -17.607152938842773, -16.2116641998291, -2.7062788009643555, 6.420594215393066, -9.332865715026855, 7.892131805419922, 3.578463554382324, -4.186956405639648, -6.379528999328613, 4.842355251312256, 3.117213726043701, -13.01279354095459, -10.951639175415039, -12.243602752685547, -11.291129112243652, -2.9333252906799316, -6.930596828460693, 12.638572692871094, -3.8964028358459473, -10.13258171081543, 10.235424041748047, -5.517358303070068, -1.9624916315078735, 12.613829612731934, -15.853497505187988, 13.669678688049316, 4.369662761688232, -9.848261833190918, -5.7454423904418945, 6.4611592292785645, 5.7685322761535645, 0.010530763305723667, 12.625744819641113], [-3.883286952972412, 12.95352840423584, -1.5731474161148071, -11.903264999389648, -12.056658744812012, 11.362407684326172, 11.455367088317871, -10.628608703613281, -20.358692169189453, 17.784669876098633, 8.36979866027832, 14.0809326171875, -2.507749080657959, -0.3779973089694977, 16.811115264892578, 11.424897193908691, 8.582475662231445, -3.9627044200897217, -9.254312515258789, 7.5351409912109375, -0.8375769257545471, 2.6458044052124023, -1.8254235982894897, -4.243372917175293, 7.888498306274414, -0.5426527261734009, 3.3414084911346436, -5.439932823181152, 5.072500228881836, -1.4409414529800415, 3.6501400470733643, -11.260086059570312, -16.224346160888672, -6.184816360473633, 19.527429580688477, -2.899843454360962, 1.8677473068237305, -21.201013565063477, 12.870220184326172, -7.301764011383057], [-25.07583999633789, -18.731246948242188, 14.111002922058105, 11.087214469909668, 14.570779800415039, -9.38914680480957, 4.799800395965576, 13.763381004333496, -6.693785190582275, -10.499795913696289, -0.8545588850975037, 5.2816386222839355, 1.4825701713562012, -3.9786934852600098, -4.5885910987854, 8.135575294494629, 12.92509937286377, -1.718226432800293, 8.694822311401367, -8.66355037689209, 2.8779399394989014, 13.10562515258789, 1.3775752782821655, -11.521336555480957, -0.2544255256652832, 5.890959739685059, -6.043610572814941, 13.136136054992676, 9.594255447387695, 2.9896295070648193, 11.904354095458984, -7.121093273162842, 7.863773822784424, 9.442808151245117, -8.601312637329102, 13.301098823547363, -21.497819900512695, -1.3004405498504639, -1.2983149290084839, -18.73502540588379]], [[-2.7920172214508057, 2.5188868045806885, -5.656531810760498, -5.830597877502441, -5.609493732452393, 13.669604301452637, -1.0325690507888794, 18.17136001586914, -5.333189010620117, -4.498459815979004, -0.9959280490875244, -7.029947757720947, -7.313581943511963, 2.2161102294921875, 10.888276100158691, -3.767721652984619, 0.3585369884967804, -4.163417816162109, -14.474526405334473, -5.749725818634033, 5.199586868286133, -14.505297660827637, 9.806446075439453, -9.870196342468262, 2.2534546852111816, 3.319092035293579, -13.73711109161377, -3.545311212539673, 6.447620868682861, 6.288435459136963, 22.363866806030273, -7.315518856048584, 7.18361759185791, 17.13975715637207, -5.091918468475342, 7.326081275939941, 0.02616034634411335, -10.770519256591797, 10.65819263458252, -8.855026245117188], [7.6438398361206055, 6.161549091339111, -21.454973220825195, 1.4506915807724, -2.153172016143799, 7.251029014587402, -13.436854362487793, 5.625545978546143, -11.453410148620605, 1.7045592069625854, -13.86951732635498, -9.553105354309082, -10.49325942993164, -14.306351661682129, 5.630703449249268, -9.517613410949707, 13.556525230407715, -11.98607063293457, -5.699799537658691, 18.694711685180664, -4.447473526000977, -12.166857719421387, 3.727604866027832, 12.335112571716309, 3.978940010070801, -0.48459944128990173, -12.235849380493164, -3.2704079151153564, -6.577786922454834, -3.027549982070923, 7.692783355712891, 6.848540306091309, -9.222820281982422, 3.5780439376831055, -3.874448299407959, -5.702003479003906, -1.7471203804016113, -2.457327365875244, -7.072585105895996, -2.86409068107605], [-5.20204496383667, -13.05034351348877, -7.849939346313477, 5.696348667144775, 10.730253219604492, 0.49254661798477173, 14.347213745117188, -20.065460205078125, 4.0158915519714355, -1.1305934190750122, 0.7141183614730835, 6.971258163452148, 14.913683891296387, 11.810667037963867, -1.0901652574539185, -2.660707473754883, 6.3765482902526855, -6.050215244293213, -11.30059814453125, 13.731684684753418, -16.976137161254883, -9.712553024291992, 9.588102340698242, -0.24542248249053955, -14.72675895690918, -12.221529006958008, 1.833414077758789, 4.663171768188477, 1.9927479028701782, 11.199687004089355, 1.9801442623138428, -8.268204689025879, 9.509221076965332, -1.351360559463501, 1.7764885425567627, -9.472442626953125, -9.290775299072266, -2.967266082763672, 7.643847465515137, 22.46989631652832], [-24.726552963256836, -13.839595794677734, -1.6149859428405762, -0.9857053756713867, -13.39706039428711, -11.482738494873047, 7.7298173904418945, -4.673069953918457, 7.6018877029418945, 3.472081422805786, 6.717984676361084, 1.4273051023483276, -0.4491289556026459, 0.378413587808609, -16.140653610229492, -9.91585636138916, 5.741928577423096, -2.3501932621002197, -2.989299774169922, -5.516500473022461, 22.206989288330078, 0.6733534932136536, -1.7205517292022705, -1.9967926740646362, -0.7494460344314575, 0.7277279496192932, 18.71070098876953, 5.001274585723877, 8.604433059692383, 15.006534576416016, -11.24831771850586, 3.027893543243408, -5.421280384063721, -2.202842950820923, 16.06165313720703, -19.39832305908203, -7.045993328094482, -9.987710952758789, -16.09815788269043, 10.323673248291016], [-11.255391120910645, 1.0063998699188232, -6.028682231903076, 0.4229124188423157, -2.248380184173584, 9.632133483886719, 17.324647903442383, -3.779935359954834, 13.165804862976074, 3.109879732131958, 15.609588623046875, 5.186041355133057, -11.772547721862793, 15.661053657531738, 12.271492958068848, -4.444497585296631, -4.271539211273193, -11.690544128417969, 4.126028537750244, 6.177190780639648, -2.7036688327789307, -20.233867645263672, 8.496999740600586, -1.9809515476226807, -16.39980125427246, 14.4105863571167, -2.257650375366211, 1.7962932586669922, 14.123616218566895, 14.26823616027832, 8.138727188110352, 17.887784957885742, -6.081530570983887, -10.031522750854492, 31.119230270385742, -3.3212013244628906, -3.344611883163452, 8.871337890625, 10.035305976867676, 12.726655960083008], [18.05599021911621, 3.7323625087738037, 10.867321968078613, -9.97049617767334, 11.095738410949707, 12.715686798095703, -10.607243537902832, 9.515283584594727, 5.779716491699219, -4.253931522369385, 4.23082160949707, -11.276490211486816, 13.362881660461426, -9.40438175201416, -8.552898406982422, 9.382611274719238, 18.09528160095215, -15.515752792358398, 2.462067127227783, 7.621337413787842, -1.8621933460235596, -0.7162049412727356, -13.65743637084961, -5.4012603759765625, 1.8335574865341187, 4.9114227294921875, 1.3293360471725464, 5.758719444274902, -7.430159568786621, 2.2415449619293213, 10.968503952026367, 17.26423454284668, -0.602013111114502, -12.193931579589844, 0.9899920225143433, 7.309290409088135, 8.839152336120605, 8.928558349609375, -19.89034080505371, -12.21589469909668], [16.892902374267578, 1.7346020936965942, 19.993101119995117, -2.5645551681518555, -4.29218864440918, -2.4359068870544434, 12.041658401489258, -8.8206205368042, -1.878051996231079, 9.91563606262207, 2.465618848800659, 0.2507580816745758, 0.49278318881988525, 10.3731689453125, 3.674471855163574, 0.7088750004768372, -5.755616188049316, 2.2768218517303467, -5.328509330749512, -22.122146606445312, -1.9095609188079834, 4.7303385734558105, -2.38419246673584, 1.924149513244629, 3.501276731491089, -2.5693840980529785, -1.1102313995361328, -3.1624832153320312, -10.898470878601074, 8.983006477355957, -8.16353702545166, -3.8810465335845947, 0.2408745139837265, 14.515548706054688, -12.119802474975586, 9.548108100891113, -10.262457847595215, 10.06336498260498, 13.084451675415039, -2.032238483428955], [-3.4884562492370605, -10.592853546142578, 20.789325714111328, 0.504776656627655, 8.127764701843262, -7.218954086303711, 4.176002502441406, -3.3072643280029297, -3.039761781692505, -11.301262855529785, 13.995169639587402, -1.867541790008545, 0.680656373500824, -8.98764705657959, -6.237151145935059, -8.180027961730957, -16.073562622070312, -2.5343234539031982, -4.322560787200928, -12.70833683013916, 19.795597076416016, 7.78623628616333, 8.977555274963379, 3.600874185562134, -6.255882740020752, -1.165549874305725, 6.887956142425537, -13.12319564819336, -19.700944900512695, -12.901256561279297, -4.204838275909424, -9.722917556762695, -7.526509761810303, 3.0097906589508057, 6.879988670349121, 16.999126434326172, -10.650553703308105, 1.7704399824142456, -6.950460910797119, -4.447211265563965], [-13.592028617858887, -19.48201560974121, 11.170894622802734, 7.527371406555176, 5.689891815185547, 3.0240702629089355, 14.715619087219238, 5.382126808166504, 2.916224479675293, -1.341091275215149, -1.407402753829956, -15.175580978393555, -5.955526351928711, 12.98100757598877, 18.096345901489258, 11.00208568572998, 1.7900511026382446, 10.801687240600586, -9.033638000488281, -9.612406730651855, 4.413445949554443, 6.247918605804443, 0.9967699646949768, 1.6269712448120117, 4.3517045974731445, -0.1543661504983902, -11.562023162841797, -2.51169753074646, -8.771551132202148, 1.1546257734298706, 8.022839546203613, -6.6454267501831055, -4.078579902648926, -9.593600273132324, -4.079387187957764, 3.200329065322876, 6.073153972625732, -1.278428077697754, 4.295156478881836, -1.0699477195739746], [-11.643330574035645, -7.706648826599121, 2.988893985748291, -23.399551391601562, -27.97711181640625, 0.5764504671096802, -7.0064568519592285, -7.17296028137207, 7.7767510414123535, -15.850931167602539, 5.332879543304443, -0.3380873501300812, -2.1667582988739014, 6.689950942993164, 13.295327186584473, 1.2359299659729004, -3.295943021774292, -25.531091690063477, -4.689576625823975, 1.52628755569458, 5.607089519500732, -3.266822099685669, 4.377586364746094, 0.8248177766799927, -4.617778778076172, 4.700962543487549, -19.487865447998047, 10.643271446228027, 5.894549369812012, -11.937814712524414, -2.7517733573913574, 0.006435263901948929, 0.1418483853340149, -20.41951560974121, 8.480985641479492, 25.029830932617188, 24.73166275024414, 20.148202896118164, 14.557599067687988, 13.226984977722168], [-10.518538475036621, -5.986112117767334, 2.8913321495056152, 3.3315823078155518, -15.861973762512207, -16.84419059753418, 20.996721267700195, -21.561105728149414, -5.030857086181641, -5.551156044006348, -30.594690322875977, 7.05126428604126, 19.063915252685547, -0.44620874524116516, 15.351759910583496, -1.5590518712997437, -1.6962536573410034, -7.1219482421875, 8.039419174194336, -4.62457799911499, 2.3275039196014404, -8.605011940002441, -5.886843681335449, -0.31780025362968445, -20.660602569580078, 6.69197416305542, -15.15007495880127, 19.23850440979004, 1.378890037536621, 9.786890983581543, -6.69191312789917, -6.13918399810791, -8.945040702819824, -6.562137603759766, 1.5066566467285156, 4.80333137512207, -15.722338676452637, 6.287245273590088, 0.3174009919166565, -5.864474773406982], [6.1011738777160645, -4.825730323791504, -3.014512300491333, -8.957810401916504, 12.073994636535645, 1.7041006088256836, 1.751871109008789, 10.88001537322998, 3.753718376159668, -1.1242798566818237, -3.239558458328247, -4.39003324508667, 12.335820198059082, 26.15422248840332, 11.678535461425781, 5.190883159637451, 6.952754974365234, 7.373882293701172, -16.046350479125977, 8.460550308227539, 8.817615509033203, -6.956700801849365, 27.809429168701172, 9.1626615524292, -15.036624908447266, -5.889461040496826, -0.9190276861190796, -6.57063627243042, 5.801612377166748, -0.021236106753349304, -25.1768856048584, 2.7094664573669434, -10.83816909790039, -5.594734191894531, -15.204473495483398, -7.201720237731934, 9.524735450744629, 5.054542541503906, 17.50037956237793, 31.174924850463867], [16.811664581298828, -8.573737144470215, 2.530322551727295, -6.343626022338867, -2.8596019744873047, 3.2262210845947266, -20.67241096496582, -16.280548095703125, 11.264191627502441, 13.72441291809082, 0.8511321544647217, 9.34242057800293, -11.750853538513184, 6.268027305603027, -9.595818519592285, -10.52902603149414, 23.772171020507812, 13.423463821411133, 11.791787147521973, -7.251529693603516, 0.44572392106056213, 6.457202911376953, -8.196968078613281, 2.2632336616516113, 15.08095645904541, 18.242435455322266, 8.886539459228516, -5.15889835357666, 17.51609230041504, -25.0848445892334, 2.7056033611297607, -5.734982490539551, 1.0149600505828857, 1.8494046926498413, 14.242558479309082, -0.8739607334136963, 5.288061618804932, 21.927490234375, 9.158326148986816, 3.8156025409698486], [-3.080444812774658, 5.79656982421875, 17.6622257232666, 16.755779266357422, 5.226832389831543, -2.105813503265381, -11.667699813842773, 15.411988258361816, 3.9245681762695312, -9.070953369140625, 6.969254016876221, 4.227279186248779, 20.089635848999023, -0.7315561771392822, -23.947120666503906, 17.86195182800293, 4.215647220611572, -11.167607307434082, 5.575564384460449, -20.171709060668945, 13.920489311218262, 15.652301788330078, 2.9976065158843994, 19.104949951171875, 6.057379722595215, -6.453042984008789, -1.4567621946334839, -13.940486907958984, -24.198381423950195, 0.286435604095459, -0.7239540815353394, 9.149018287658691, -1.6665289402008057, 10.005156517028809, -1.4471622705459595, -1.9040485620498657, 6.068484306335449, -21.175785064697266, -0.1716778725385666, -5.920670032501221], [-2.7720024585723877, 2.320509195327759, 11.0137939453125, -19.401615142822266, -8.60557746887207, -5.626891136169434, -2.050389528274536, -14.561871528625488, 4.498722553253174, 13.336830139160156, 9.61351490020752, -2.046936273574829, 3.8772504329681396, -18.800514221191406, -27.485567092895508, -7.979882717132568, -1.4322963953018188, -1.241417407989502, -2.354674816131592, -7.594590663909912, -20.574352264404297, 9.908276557922363, 2.155386447906494, 0.512703001499176, 11.69627857208252, 11.078804016113281, -4.3009724617004395, -13.014177322387695, -0.2709621489048004, -4.757235050201416, -3.746363639831543, 13.671876907348633, 5.494434356689453, -3.2922115325927734, 9.149206161499023, 4.156543254852295, 4.894996166229248, 10.803900718688965, -9.495052337646484, -12.791464805603027], [-12.745451927185059, -3.884885549545288, 1.8076399564743042, -3.639744281768799, 5.819795608520508, -3.555231809616089, 11.125383377075195, 7.7765703201293945, -16.48996925354004, 16.233646392822266, -8.041276931762695, -1.951936960220337, 4.128253936767578, -1.3826448917388916, 36.742897033691406, -3.715115785598755, 3.7052347660064697, 26.44293975830078, 10.661900520324707, -20.75322914123535, -1.1614481210708618, -10.211413383483887, -25.889537811279297, -11.997297286987305, -1.3282406330108643, 24.658815383911133, 8.618520736694336, 16.29888343811035, -1.8841197490692139, -11.249520301818848, 2.065704584121704, 26.533794403076172, -27.164600372314453, -4.436773300170898, 8.330897331237793, -14.566864013671875, -18.101417541503906, -11.48182201385498, -9.04172134399414, 6.4486188888549805], [0.6578389406204224, 7.145251274108887, -11.25394344329834, 17.23259735107422, -12.018904685974121, 6.492214202880859, 0.21072795987129211, 0.2704642415046692, -15.641800880432129, -9.83074951171875, -1.7235713005065918, 15.487439155578613, 12.082982063293457, 11.31146240234375, 0.27821844816207886, -4.558160305023193, 12.398781776428223, 4.655138969421387, 6.608592987060547, 14.062520027160645, 19.48325538635254, 7.266186237335205, 18.558109283447266, -6.632297992706299, -6.226136684417725, -7.018399238586426, 3.040802240371704, 1.654086947441101, 1.3903886079788208, -10.307689666748047, 14.110692977905273, 13.797057151794434, -5.993381500244141, 3.933081865310669, 6.204126358032227, -11.787880897521973, 3.228605031967163, -11.436985969543457, -12.628621101379395, 16.25893211364746], [-12.617246627807617, 1.1610698699951172, 1.166131854057312, 4.401388168334961, 0.7359312176704407, -2.056354284286499, 4.66123104095459, -17.444482803344727, -2.9064719676971436, 5.513473033905029, 4.257665634155273, -4.46103572845459, 3.4373862743377686, 0.656114935874939, -0.2523054778575897, -5.371763229370117, 27.068363189697266, -0.09362483024597168, 12.222675323486328, -24.2115535736084, -7.797802925109863, 13.89868450164795, -6.805019855499268, -2.2671000957489014, -3.596085548400879, 5.299811363220215, -0.6138863563537598, 18.5035343170166, 5.387520790100098, 8.83900260925293, 14.384583473205566, -11.00790023803711, -3.629953145980835, -0.5528726577758789, -2.212418556213379, 6.3443732261657715, 7.549615383148193, 16.387449264526367, -5.7290215492248535, -16.932233810424805], [9.676362991333008, -6.672552585601807, 0.6706132292747498, -22.982913970947266, 19.517126083374023, 6.361164093017578, -2.818260669708252, 9.810026168823242, -5.92804479598999, 2.65749454498291, 18.79962730407715, 18.051467895507812, 10.45858097076416, -1.7227935791015625, -16.680387496948242, 3.9748239517211914, 14.428180694580078, -13.370321273803711, -17.496856689453125, 12.924304008483887, 2.9984781742095947, -29.064579010009766, -8.682890892028809, -4.8866071701049805, -15.054000854492188, 4.076461315155029, 17.06798553466797, -11.739205360412598, -9.613726615905762, 19.704174041748047, 3.0648598670959473, 31.476083755493164, 4.617617130279541, -1.9313561916351318, 6.129884243011475, 11.661703109741211, -7.424617290496826, 3.915637731552124, -8.141901016235352, 8.28292179107666], [2.254405975341797, 7.41323709487915, -10.487846374511719, -10.494134902954102, 14.276270866394043, 20.931720733642578, -11.828481674194336, 16.106914520263672, 1.567952036857605, -5.409371852874756, 7.736701965332031, 17.889719009399414, -17.10649871826172, 2.6297178268432617, -12.782610893249512, -2.2114298343658447, -4.909976482391357, -13.983146667480469, 18.670827865600586, 0.6828560829162598, -9.074273109436035, -4.5089802742004395, -1.6283557415008545, 4.157289028167725, 15.309833526611328, 9.100821495056152, 8.162029266357422, 8.07099723815918, -2.0803210735321045, -5.35870885848999, -0.05800304189324379, -7.296660900115967, -2.9267995357513428, 24.152244567871094, -6.7763142585754395, -15.849661827087402, 4.5212507247924805, 7.557304382324219, 3.47771954536438, -9.064870834350586], [17.576642990112305, 12.994094848632812, -8.99040699005127, 11.475059509277344, -14.635259628295898, -8.291335105895996, 10.344769477844238, -4.8813323974609375, -8.271539688110352, -6.455207347869873, -15.429548263549805, 9.257062911987305, 3.73610520362854, 0.1469845175743103, -6.325124740600586, 4.9616217613220215, 19.35028648376465, 9.35000991821289, -3.837404251098633, -5.734892845153809, -18.078853607177734, -5.89427375793457, 18.4991397857666, -17.850948333740234, -14.225772857666016, 5.425477027893066, -8.940192222595215, -1.9700177907943726, 3.595914602279663, 11.904916763305664, 14.761319160461426, 7.433014869689941, 2.360696315765381, -6.091387748718262, 7.966833591461182, -0.8501905202865601, -6.114641189575195, -3.068824052810669, -14.808286666870117, -13.68200969696045], [15.844869613647461, -2.633329153060913, -13.286117553710938, 14.690669059753418, 19.583805084228516, 5.252495765686035, 6.156156063079834, -17.261812210083008, 2.328601598739624, 2.6871519088745117, -9.455240249633789, -9.334071159362793, -11.902397155761719, 6.325799465179443, 16.55535125732422, -0.2847888171672821, 21.980377197265625, -16.308626174926758, 7.147759437561035, 4.48346471786499, 7.424723148345947, -5.515527725219727, -1.9807329177856445, 11.278165817260742, 7.54581356048584, 0.24858812987804413, 5.51538610458374, 1.8318424224853516, 3.1767656803131104, 7.195338726043701, 1.3862718343734741, 0.5399742126464844, 14.6245756149292, -4.6442766189575195, -2.4051785469055176, -0.7483158111572266, 13.084339141845703, 1.1770142316818237, 13.251119613647461, 7.409710884094238], [4.665495872497559, -18.385475158691406, -3.0955307483673096, -17.19036293029785, 2.723568916320801, -10.762625694274902, -3.7436699867248535, 0.7341000437736511, -6.1006951332092285, 2.395148754119873, -1.9197793006896973, 0.0022626735735684633, -16.825056076049805, -11.115426063537598, 20.31688117980957, -11.552689552307129, 12.558515548706055, 12.163823127746582, 10.587538719177246, -9.121946334838867, -1.845466136932373, 2.2142534255981445, -13.12086296081543, 18.409534454345703, 10.025553703308105, 10.98180103302002, -11.839444160461426, 12.705652236938477, 15.944979667663574, 22.12776756286621, 2.292893409729004, -14.113723754882812, -8.956380844116211, 2.642737865447998, -17.5058650970459, -13.832036018371582, -7.830338478088379, -1.0581034421920776, 0.009136682376265526, 5.814276695251465], [-0.9848525524139404, 4.935464382171631, 5.34675407409668, 27.857961654663086, 13.731912612915039, -18.61145782470703, -4.231020927429199, -7.604012489318848, 8.336804389953613, -1.3481924533843994, 0.5843759179115295, -17.856708526611328, 13.200114250183105, 7.457170486450195, -5.481003284454346, 17.754289627075195, 15.935869216918945, -5.584973335266113, 0.22260455787181854, -10.840410232543945, 1.311660647392273, -3.107696294784546, -7.851988315582275, 13.848828315734863, -8.391225814819336, 17.228036880493164, 8.145947456359863, 18.30923080444336, -17.474912643432617, 27.664417266845703, 3.9679114818573, 13.565340995788574, -21.170242309570312, -11.942749977111816, 14.246493339538574, 14.869353294372559, 1.6023879051208496, -12.738227844238281, -2.603264331817627, 0.03234068676829338], [4.9418110847473145, 12.455872535705566, 7.53629207611084, 5.066662311553955, -15.11407470703125, -13.038989067077637, 5.930056095123291, 6.522916793823242, 11.34975528717041, 9.247902870178223, -0.7689853310585022, 21.94053840637207, 1.3962287902832031, 14.650774002075195, -2.0369644165039062, 9.560665130615234, 17.980554580688477, -5.307731628417969, -3.9858920574188232, -12.337777137756348, 2.294814348220825, -5.5605034828186035, -17.282625198364258, -0.29698997735977173, 3.4007840156555176, 9.46576976776123, 10.92447566986084, -3.3573110103607178, -18.752479553222656, 10.443963050842285, 9.3143949508667, -0.7647600173950195, -25.56262969970703, 8.061532020568848, 16.68867301940918, 8.844194412231445, -10.22253704071045, -8.011916160583496, -6.048654079437256, -22.460786819458008], [2.312779664993286, 1.5309032201766968, -21.979671478271484, 12.630707740783691, -6.757887363433838, 2.431974172592163, -0.37570804357528687, 6.259533882141113, 5.572890281677246, -6.2976603507995605, -25.438045501708984, 7.475685119628906, 7.240503311157227, 0.10903394222259521, 0.8783815503120422, -7.60658597946167, 4.013547420501709, -17.02809715270996, 6.949894905090332, -5.1241068840026855, -0.9750019311904907, -9.984272956848145, -8.359432220458984, -12.001249313354492, 26.200702667236328, -4.9810051918029785, 3.3739640712738037, -18.143108367919922, 8.795186042785645, 18.84624481201172, -4.442968845367432, 17.180883407592773, 2.233562707901001, 10.515603065490723, 16.90141487121582, -33.63865661621094, 4.188434600830078, 1.647464632987976, 25.658706665039062, 9.281136512756348], [-9.248353004455566, 4.45302152633667, 36.7160758972168, -16.66054916381836, 18.0421142578125, 5.616891860961914, 9.045633316040039, 7.248340606689453, 3.878925323486328, -6.854955196380615, -4.764349460601807, 13.912285804748535, 5.690921306610107, 8.911282539367676, 11.819814682006836, 12.450787544250488, -4.7096381187438965, -17.994421005249023, 0.13295388221740723, 13.508182525634766, -9.3527193069458, 3.037538766860962, -2.286776065826416, 18.40961265563965, -3.1549670696258545, 1.4644001722335815, -16.380264282226562, -0.45623090863227844, 6.9117560386657715, 10.234349250793457, 5.524405002593994, 7.81976842880249, -14.147621154785156, -0.9286944270133972, 8.992328643798828, 10.64216423034668, 0.034336451441049576, -6.48018741607666, 13.537233352661133, 3.6411309242248535], [14.76453685760498, 8.751717567443848, -8.791592597961426, 8.099320411682129, 5.057187557220459, 8.222576141357422, -1.447127342224121, 0.705395519733429, -4.548584938049316, -3.506169080734253, -12.281312942504883, -3.6272037029266357, 4.709390640258789, 11.58312702178955, 8.917158126831055, 8.014824867248535, 2.336479663848877, 2.3997390270233154, 3.783385992050171, -1.1797057390213013, -0.39523202180862427, 7.779631614685059, 12.770581245422363, 7.680374622344971, -8.674774169921875, 19.317604064941406, -11.565452575683594, -5.449926853179932, -15.478025436401367, 12.441034317016602, -8.773233413696289, 13.360029220581055, 12.538923263549805, 11.631139755249023, -10.185489654541016, -20.1593074798584, 7.736968517303467, -5.296000957489014, -4.921814918518066, 3.537716865539551], [-22.813819885253906, -3.8620717525482178, 1.154964804649353, 8.213058471679688, -17.662860870361328, 2.448753595352173, 1.2304542064666748, -5.91050386428833, 22.192806243896484, 15.125837326049805, 25.831567764282227, -3.178351879119873, 2.7858712673187256, 4.2511773109436035, -4.838488578796387, -2.9231443405151367, -1.0464850664138794, 42.464210510253906, -3.2014682292938232, 16.783832550048828, 7.884583950042725, 3.851715564727783, 19.347713470458984, -11.0158109664917, -13.857521057128906, 1.5614643096923828, 1.7524222135543823, 23.856271743774414, -0.9439356327056885, -20.83030128479004, -11.33996295928955, 11.419421195983887, -13.630243301391602, 0.6388075947761536, 14.265213966369629, 3.7618236541748047, 1.2310813665390015, 2.6164309978485107, -12.355768203735352, -10.652420997619629], [0.17879486083984375, 4.022541046142578, -2.0733048915863037, 23.673961639404297, -2.5327253341674805, 3.412778854370117, -12.872211456298828, -17.67511749267578, -4.322973728179932, -3.5294079780578613, 6.882569313049316, 21.88640785217285, -8.582916259765625, 17.408132553100586, -4.4226179122924805, -5.6549177169799805, -2.481689453125, 5.346524238586426, -7.405989170074463, -27.877241134643555, -0.5923110246658325, 9.392906188964844, 13.216519355773926, -7.104317665100098, 10.789806365966797, 13.388591766357422, 24.928327560424805, -1.306678295135498, -21.53531837463379, -5.982400894165039, -9.113534927368164, 17.198453903198242, -3.134037733078003, 5.986979007720947, -10.59813404083252, -6.38908576965332, -0.1308409571647644, -0.515247106552124, 4.661383152008057, -5.83941125869751], [-0.301842600107193, -21.190778732299805, -10.309266090393066, 10.166136741638184, 16.483291625976562, 28.724517822265625, 11.208542823791504, 0.9846205115318298, -14.59162425994873, 2.182831048965454, 0.8599768280982971, 16.524600982666016, 15.601031303405762, 1.2181133031845093, 4.661952495574951, -17.306894302368164, 0.9842881560325623, -13.759334564208984, -5.743084907531738, -13.454755783081055, 18.02684211730957, 2.793724298477173, 3.926569700241089, 11.92443561553955, 1.406947374343872, 8.832501411437988, -10.172725677490234, -18.468229293823242, -15.515435218811035, -8.464917182922363, -0.8286988735198975, -3.4460859298706055, -9.279892921447754, 8.398162841796875, -8.716355323791504, -1.4240466356277466, 16.07830810546875, -0.6637853980064392, 1.0145387649536133, -1.5063952207565308], [0.35492396354675293, 12.73103141784668, -4.879940509796143, 8.388208389282227, -18.65083122253418, -10.642510414123535, -22.221603393554688, -4.9219651222229, 3.8679044246673584, 0.19303961098194122, -2.6977152824401855, -8.710144996643066, 11.813241004943848, -4.562170028686523, -12.75861930847168, 2.7182908058166504, 6.92959451675415, -6.732588291168213, 0.688647985458374, 21.235612869262695, 10.152048110961914, 10.46204662322998, 2.2005808353424072, 16.600345611572266, -2.8656678199768066, -12.420991897583008, 9.68856430053711, 2.297605276107788, -12.787787437438965, 9.695181846618652, -12.798359870910645, 1.4210588932037354, 18.031999588012695, 13.887372016906738, 5.429744720458984, -3.5713133811950684, 3.9151010513305664, -2.695099353790283, 3.7451510429382324, 12.003401756286621], [-1.622498869895935, -4.1234049797058105, 2.5369880199432373, 3.198843479156494, -17.958322525024414, -13.579771995544434, -11.00542163848877, -22.32487678527832, -4.905346870422363, 0.507064938545227, -11.448617935180664, -2.8942251205444336, -13.63717269897461, 42.17399215698242, 2.2857298851013184, -16.783321380615234, 3.021287441253662, 11.759373664855957, 7.380692958831787, -6.149994373321533, 4.136993885040283, -8.405577659606934, 9.812567710876465, -15.049147605895996, 13.9190034866333, 8.889862060546875, -29.413227081298828, 0.8082304000854492, 10.058119773864746, 3.636121988296509, 6.655051231384277, 7.024346828460693, -6.732055187225342, 13.02916145324707, 3.7097673416137695, -7.194023609161377, 4.205528259277344, 29.717647552490234, 3.7637126445770264, -7.367237567901611]], [[5.605090141296387, 2.8375749588012695, 2.207627534866333, 13.632076263427734, 5.5544753074646, -7.834481716156006, 1.3915098905563354, -0.8457843065261841, 5.024908065795898, -2.741109609603882, 5.081872463226318, -0.19166222214698792, 5.72725772857666, 3.334911584854126, 6.320756912231445, -13.294587135314941, 5.815276622772217, 12.691202163696289, 7.644976615905762, 0.3377940058708191, -6.526814937591553, -3.190467357635498, -7.840524673461914, -13.22456169128418, 6.559296131134033, 1.2987276315689087, 3.850297689437866, 8.345749855041504, 3.2632384300231934, -1.61122465133667, -6.897652626037598, -5.204042911529541, 7.285632133483887, -6.953718662261963, -1.1404531002044678, -8.523665428161621, -1.0327019691467285, -6.061590671539307, 4.847927570343018, -4.251469612121582], [1.0814967155456543, -2.14994215965271, -4.841971397399902, -2.843017578125, -0.3459499776363373, -1.589756965637207, 0.7752954363822937, -13.984031677246094, 4.219691276550293, -4.971004009246826, 7.297010898590088, 4.2327775955200195, -11.61880111694336, 6.460246562957764, -4.490274906158447, 1.8024369478225708, -1.3977224826812744, -5.0779948234558105, 0.5833950042724609, 0.7026094198226929, -11.979598045349121, 0.5380834341049194, -3.4506783485412598, -3.811316967010498, 12.724726676940918, -0.11497899144887924, -15.05551528930664, 1.8068084716796875, 4.890921592712402, 0.15988600254058838, 2.652778387069702, 2.1829562187194824, 6.813582897186279, -6.568450927734375, 1.0140221118927002, -0.28714072704315186, -1.4860676527023315, 5.1208882331848145, 2.113308906555176, -3.4916653633117676], [-1.545949935913086, 5.905976295471191, -1.216016411781311, 11.718343734741211, 2.7990615367889404, -7.431894302368164, 6.603961944580078, -6.554729461669922, 3.4490175247192383, -14.654425621032715, 2.510145664215088, -4.0246171951293945, 9.51518440246582, 13.972078323364258, 11.929403305053711, 0.5972638130187988, 0.053252123296260834, 6.368650436401367, -1.9328358173370361, -1.4847930669784546, -0.6078795790672302, -2.003237009048462, -7.942484378814697, -2.9308221340179443, -8.31383991241455, 2.068356990814209, -0.0984358862042427, -0.12480419129133224, -2.936563491821289, 7.123785972595215, 0.4985950291156769, -4.837316036224365, 9.52857780456543, 0.38778600096702576, -0.7293943166732788, -6.392073631286621, 2.140787124633789, 0.16388358175754547, -3.5446107387542725, 8.631171226501465], [10.02003288269043, -5.672560214996338, 9.466050148010254, -0.3579899072647095, 14.873824119567871, 6.87276029586792, 7.205870628356934, 13.347668647766113, -10.637020111083984, 3.7861063480377197, 4.635223388671875, 3.652592182159424, 11.141427993774414, -3.0275020599365234, -7.50087833404541, -5.970913410186768, 6.04745626449585, 4.568042755126953, -13.373466491699219, 1.4521554708480835, 0.05182139575481415, 9.683969497680664, -1.7914048433303833, 2.0616836547851562, -3.3993282318115234, 2.8823962211608887, 7.493058681488037, 3.7696938514709473, 3.2212166786193848, 1.6408140659332275, -14.69714069366455, 2.1143765449523926, 0.39273494482040405, -4.393840312957764, 3.63637638092041, 1.5113228559494019, 7.166411399841309, 12.131978034973145, 0.14153027534484863, 2.838782787322998], [13.82416820526123, 1.9220595359802246, -1.5569744110107422, 8.866899490356445, -0.36562445759773254, -10.754650115966797, 2.9064507484436035, 2.3133647441864014, -1.7432715892791748, 6.822172164916992, -0.7959820032119751, 0.9727423787117004, 1.192012906074524, 5.515224456787109, 7.252913951873779, 3.9384925365448, 4.747757911682129, 8.079634666442871, 5.9621968269348145, -1.0315219163894653, 4.907844543457031, -10.86563777923584, 2.0982885360717773, -7.078189373016357, -7.629854679107666, -11.686037063598633, 0.01123839057981968, 0.7165573239326477, 4.465212345123291, 0.1195351779460907, 5.55266809463501, -7.743391513824463, 0.7949284911155701, 3.2805612087249756, 8.745970726013184, -7.6570892333984375, 0.8143393397331238, -7.181004524230957, 14.94677448272705, 1.0979129076004028], [5.725800037384033, 5.219231605529785, 11.113604545593262, -6.244916915893555, 1.9213314056396484, 7.738753795623779, 1.7401007413864136, 21.069120407104492, -16.57404136657715, -5.239253520965576, 4.19734001159668, 10.94790267944336, -1.415935754776001, -2.7472944259643555, -10.611207008361816, 3.4134061336517334, -3.978208303451538, -8.980677604675293, -6.313796520233154, 0.0017128337640315294, -6.363679885864258, 5.1469011306762695, 2.061890125274658, -2.4959726333618164, 1.5894699096679688, 1.3472728729248047, 0.9316365122795105, -7.064223289489746, -5.193385124206543, 0.4599294662475586, -3.618440866470337, 9.638349533081055, -10.521904945373535, -4.812180042266846, 5.568998336791992, 2.090177059173584, 2.271162271499634, 16.30289649963379, 7.060150623321533, -9.114815711975098], [4.868047714233398, -8.817227363586426, -3.421823501586914, -3.039045572280884, -2.2999892234802246, -1.9567991495132446, 3.8348164558410645, -8.12447452545166, -11.26236629486084, -0.5392096638679504, -9.888420104980469, -0.23469796776771545, 4.592309474945068, -4.809593677520752, 1.7661527395248413, 16.857236862182617, 0.3523874580860138, 7.807337284088135, 2.0705716609954834, -7.328507423400879, -6.659804821014404, -6.13342809677124, -4.066315650939941, -8.749874114990234, -2.880859613418579, 3.9465744495391846, 1.1991750001907349, 9.895371437072754, 4.570706367492676, -6.760227203369141, 11.597599983215332, 3.092176675796509, -6.052323818206787, 3.482099771499634, 2.6485936641693115, -0.870039165019989, -4.141269683837891, -10.169058799743652, -4.482013702392578, -8.923431396484375], [-13.413372039794922, -10.672054290771484, 6.504923343658447, -2.5867626667022705, 20.186864852905273, 9.343244552612305, -5.656979084014893, -3.6240034103393555, 7.903703212738037, 1.3961166143417358, -7.715353965759277, -4.94071626663208, -3.0970778465270996, 5.29831075668335, 4.578401565551758, -4.978806972503662, 7.029195785522461, 3.360551595687866, 4.482120037078857, -2.5008468627929688, 6.96057653427124, 3.8050854206085205, -8.498061180114746, -0.09376230835914612, -4.112637042999268, -6.30267858505249, 4.8118414878845215, 7.952079772949219, -4.409711837768555, 3.4765303134918213, 1.5917284488677979, 7.40867805480957, 5.830107688903809, 1.5359827280044556, 4.031250476837158, 17.28777503967285, 0.8110796213150024, 15.336151123046875, 9.782511711120605, -2.4522945880889893], [-4.282162189483643, -0.9089085459709167, -3.1934521198272705, -8.148737907409668, -5.249626159667969, -5.955532550811768, -0.5837966203689575, -0.9055155515670776, -19.624298095703125, -3.014535665512085, 1.5425834655761719, -2.320416212081909, -3.5846948623657227, -1.0593924522399902, -1.4302096366882324, 11.194759368896484, -2.0372867584228516, -9.493234634399414, 0.39288991689682007, -5.695562362670898, -4.521279335021973, -5.7316131591796875, 5.855637073516846, 1.5447015762329102, 4.345414638519287, 2.631375551223755, -7.835904121398926, -6.862718105316162, 5.641930103302002, -8.78503131866455, 19.335384368896484, 2.6466169357299805, -6.618673801422119, -0.9222617149353027, -0.31165844202041626, -9.938192367553711, -5.610417366027832, -1.3295342922210693, -2.053905487060547, 6.179689884185791], [4.1780219078063965, 3.778900384902954, 7.82230281829834, -11.432904243469238, -4.865716934204102, 5.042136192321777, -0.5267088413238525, 9.991680145263672, 6.234838485717773, 5.908017635345459, -11.370128631591797, 8.508825302124023, -7.718845367431641, -11.342247009277344, -12.836424827575684, -2.1439549922943115, -6.558899402618408, -7.384721755981445, -5.8340864181518555, -2.865316152572632, -0.3442893326282501, 6.19883918762207, -2.994624376296997, -6.202577590942383, -6.393644332885742, 3.355759382247925, 5.9558868408203125, 10.084686279296875, -8.33035659790039, -1.7108350992202759, -14.5048189163208, 3.0453741550445557, 5.331453323364258, -13.004671096801758, -0.6607310175895691, 10.376466751098633, -2.1348562240600586, 5.713566303253174, 1.1687999963760376, -5.21407413482666], [3.770556926727295, -1.6948843002319336, -1.3496513366699219, 4.349684715270996, 1.123813271522522, 1.3516942262649536, 10.827099800109863, -0.8147067427635193, 11.67760944366455, 11.035857200622559, -2.679636001586914, 11.518383979797363, 1.6064430475234985, 1.09208345413208, 1.0392470359802246, -11.679465293884277, 0.6876648664474487, -1.192121982574463, 10.107840538024902, 0.7553310394287109, -0.054907940328121185, 3.8581104278564453, -14.71760368347168, -2.287923812866211, 4.956911087036133, 6.735642433166504, 3.2934343814849854, 7.635201454162598, -1.739776372909546, -1.646004557609558, -1.2844375371932983, -5.227696418762207, 15.000850677490234, -12.611617088317871, 8.468780517578125, 13.337616920471191, 5.075786113739014, 12.639019012451172, 0.0754413828253746, -11.778319358825684], [-6.243310451507568, 4.66636848449707, 2.2020111083984375, 8.970257759094238, -8.793195724487305, 0.3091616630554199, 0.3090921938419342, 8.18575382232666, 2.5851902961730957, 0.5181176066398621, -3.8638417720794678, -0.06638404726982117, 12.142603874206543, -6.2019453048706055, -8.461477279663086, -14.613593101501465, 10.18360424041748, -2.9627764225006104, 10.6248197555542, 4.233487129211426, 2.526515007019043, -8.304322242736816, 1.2278722524642944, 0.7755540609359741, 2.9253082275390625, -8.533486366271973, 13.217795372009277, 2.0922181606292725, 3.874913215637207, -0.4871272146701813, -1.418885350227356, -3.1684985160827637, 0.7622815370559692, 6.781551837921143, 15.260984420776367, -10.21323013305664, -3.831209659576416, -8.057413101196289, 6.299349784851074, -4.789897441864014], [-9.556801795959473, 7.058039665222168, 5.697274208068848, 7.7716569900512695, -3.826713800430298, -3.076444149017334, -6.466460227966309, 0.07953857630491257, 4.679760932922363, 0.6393372416496277, -5.942823886871338, -3.2993886470794678, -0.576863169670105, 2.2863831520080566, -5.248497486114502, 5.344490051269531, 1.4126269817352295, -5.631530284881592, 6.152811527252197, 6.450500965118408, 2.360537052154541, -10.41448974609375, 10.18371868133545, 7.355844974517822, -8.540386199951172, -3.3755898475646973, -0.014606865122914314, 1.2929414510726929, -4.178191661834717, 2.56901216506958, 3.36747407913208, 4.431427001953125, -7.752849578857422, 6.188289165496826, 8.075736045837402, -1.4173824787139893, -2.0339701175689697, -3.7042343616485596, 1.5706390142440796, -1.0349856615066528], [-4.336208820343018, -9.698701858520508, 0.4325096011161804, -12.98735523223877, 5.3722968101501465, 1.3491660356521606, -11.282705307006836, 10.31186294555664, -2.1997175216674805, 1.354130506515503, -10.476057052612305, -7.074218273162842, -0.8627490997314453, -9.3906831741333, -2.358975887298584, 5.324561595916748, -5.522848129272461, 0.7365341782569885, -7.2542405128479, -0.30734431743621826, 8.684110641479492, 3.192917823791504, 1.7001049518585205, -3.2962584495544434, 3.92289662361145, -7.945718765258789, -2.846705675125122, 10.132120132446289, -8.074200630187988, 2.625120162963867, -10.678805351257324, 11.427873611450195, -4.459354877471924, 3.508730411529541, -8.20495891571045, 19.15080451965332, -5.833855152130127, 0.19168218970298767, -4.161949157714844, -2.4243240356445312], [8.585366249084473, 3.66104793548584, 3.5248260498046875, -5.214276313781738, 0.039550457149744034, -7.857756614685059, -0.29795345664024353, -11.890909194946289, -0.03520672023296356, -3.629031181335449, -4.944485187530518, 7.900670051574707, -17.985506057739258, 1.691463828086853, 1.2629177570343018, 5.059161186218262, -12.332923889160156, 2.568769931793213, 7.400146961212158, -13.905461311340332, 9.79649543762207, 2.6007583141326904, -2.634956121444702, -12.628975868225098, -3.9483425617218018, -4.892479419708252, 6.361608505249023, -6.60288667678833, -5.986365795135498, 7.592726707458496, 4.461187839508057, -1.5566613674163818, 10.489999771118164, 5.739281177520752, -0.722607433795929, -0.2903624475002289, 8.514415740966797, 6.251467227935791, -2.8046281337738037, -1.7325096130371094], [4.6014084815979, 2.3571648597717285, -8.207799911499023, -6.96290397644043, -12.658088684082031, 0.12461615353822708, -5.725515842437744, 8.233586311340332, 5.436157703399658, -7.949267864227295, 2.1685574054718018, -2.0481770038604736, -13.134102821350098, 0.32467278838157654, -5.509378433227539, 4.762791156768799, 6.023326873779297, 1.5382044315338135, -6.009839057922363, 3.41343355178833, 4.195707321166992, 5.113497257232666, 16.61689567565918, 10.851095199584961, -11.034393310546875, -3.4190616607666016, 1.5331624746322632, -9.270282745361328, -1.3891547918319702, -0.8234393000602722, -1.086696743965149, -3.173726797103882, -6.5855488777160645, -2.121782064437866, -7.588773727416992, -3.6811416149139404, -3.1079487800598145, -7.528091907501221, 0.5395172834396362, 16.92584228515625], [7.0308685302734375, -8.017342567443848, 0.0073569961823523045, 0.013171076774597168, -5.932043075561523, 1.4457476139068604, 7.021915912628174, -0.15831232070922852, -6.664730072021484, 6.671461582183838, 9.425042152404785, 1.950015902519226, 3.7287933826446533, 4.931870937347412, 7.495665550231934, -12.628917694091797, 3.7904374599456787, 5.866946220397949, -7.721940994262695, -7.420769214630127, 6.563211917877197, 6.925429344177246, -4.2525224685668945, 8.069293975830078, 4.385097980499268, 1.6141752004623413, -2.0934360027313232, -7.606149673461914, 1.7878886461257935, -7.0422282218933105, 2.9699208736419678, -5.415953636169434, 6.106629848480225, 0.27246788144111633, -0.6058081984519958, 3.2331573963165283, 1.2844767570495605, 5.79665470123291, 1.1908354759216309, 5.1379499435424805], [5.062669277191162, 0.6657841801643372, -0.6290664672851562, -9.73733139038086, 0.3014403283596039, 10.117449760437012, 8.018876075744629, -2.5043258666992188, -4.010373592376709, 4.435190677642822, -3.1706490516662598, -0.8781995177268982, -9.374924659729004, 11.68345832824707, -2.33355975151062, 6.3181939125061035, 5.286464214324951, 4.663014888763428, 5.1509809494018555, -12.285604476928711, 9.373037338256836, 3.6212823390960693, -2.089184522628784, -4.284064292907715, -7.288200855255127, 3.8760902881622314, -5.185003280639648, -2.918325901031494, 1.0337512493133545, 7.322022438049316, 2.1167960166931152, -4.19301176071167, -20.27003288269043, 3.6036739349365234, -0.07431406527757645, 4.405587673187256, 5.249274253845215, -4.060348033905029, -4.050785541534424, -3.1556921005249023], [0.8423362970352173, 6.4199137687683105, 4.337412357330322, 2.9825949668884277, 10.554923057556152, -8.145160675048828, 0.6477647423744202, -8.91053295135498, 3.596672773361206, 5.937814235687256, 3.3540940284729004, -2.4639670848846436, 9.628073692321777, -0.11905498802661896, 0.3993825316429138, -6.300082683563232, 2.042442798614502, -6.515522003173828, 6.720029354095459, 7.719483375549316, 1.423275113105774, -6.107259273529053, 0.23928888142108917, -5.881597995758057, -2.131267547607422, 0.3706473708152771, 1.6357096433639526, 11.795510292053223, 10.163416862487793, -10.601052284240723, -3.98469877243042, 1.8006442785263062, 8.891701698303223, 1.7230892181396484, 1.4664360284805298, 4.1526570320129395, 4.352785587310791, 3.809138536453247, 2.499133825302124, -4.382589817047119], [1.2696223258972168, -3.4481029510498047, -8.513006210327148, -4.519545078277588, 12.263396263122559, 8.553254127502441, -0.22337718307971954, -6.539067268371582, -5.57443380355835, -6.041659832000732, -6.199957847595215, 1.0887086391448975, 3.9137184619903564, 1.4896479845046997, 1.897339940071106, 3.616895914077759, -4.625697612762451, -1.1393307447433472, -7.137940883636475, -3.335541248321533, -12.36662483215332, 2.563074827194214, -3.2853543758392334, -3.643127679824829, -2.4156272411346436, 14.850081443786621, -1.0802111625671387, -2.9210338592529297, -0.2216455489397049, -6.186795234680176, -0.6736673712730408, 4.426939487457275, 2.3934459686279297, -8.350871086120605, -5.665552139282227, 4.006333351135254, 2.151597499847412, 1.6051158905029297, 0.27352234721183777, 2.284461259841919], [-0.58870530128479, -6.532809734344482, 5.276821613311768, 4.779202461242676, -2.570467472076416, 5.6974711418151855, 2.1779673099517822, -8.904215812683105, -5.88136625289917, 9.949417114257812, 8.867582321166992, 5.320309162139893, -12.043222427368164, 7.93834114074707, 6.990289688110352, 14.420827865600586, -3.6220266819000244, -12.071845054626465, -9.686176300048828, 3.611328125, -5.0936360359191895, -10.507652282714844, 4.931914806365967, -12.736145973205566, 1.545230746269226, 8.722565650939941, -12.337883949279785, -8.344379425048828, 3.159104824066162, -4.6794939041137695, 2.8827290534973145, -5.4365010261535645, -3.811077356338501, -1.785578966140747, -13.202760696411133, -9.791519165039062, -4.166419506072998, -7.750210762023926, -0.6379286646842957, -5.184017181396484], [0.7262675166130066, 3.9861721992492676, -2.837839365005493, -1.8093864917755127, 9.990147590637207, 2.9938151836395264, -2.391249418258667, 4.904812335968018, -0.6725709438323975, -8.17956256866455, -5.974118709564209, -8.101810455322266, 5.3418498039245605, 3.0269336700439453, -5.887036323547363, -6.477940559387207, 1.1547279357910156, 2.853747606277466, -3.075965404510498, 1.601863145828247, -0.15290619432926178, 1.2116584777832031, 4.3944525718688965, 11.994668006896973, -2.296658992767334, -1.5025124549865723, 1.7406119108200073, 0.4792991280555725, -2.6069724559783936, -3.533560037612915, 9.818294525146484, -3.3935678005218506, -10.924782752990723, -1.4944157600402832, 6.803563117980957, 2.260874032974243, 3.9163882732391357, 0.7656297087669373, -8.83454704284668, 6.755600929260254], [3.3808863162994385, 8.45352554321289, 0.17620563507080078, -4.480461597442627, -8.65449047088623, -3.4484612941741943, -3.4822447299957275, -6.9931159019470215, 6.622045516967773, -0.41741934418678284, -3.5339508056640625, 1.322805404663086, -4.380621433258057, -0.8134689331054688, -15.4256591796875, -9.677785873413086, 2.919668197631836, 5.75147008895874, -10.210587501525879, 9.519973754882812, -4.580174446105957, 6.491475582122803, -2.331848382949829, -9.572623252868652, -2.459773063659668, 11.738612174987793, 2.6699941158294678, -2.508509635925293, -6.268092155456543, -4.001835346221924, 9.917546272277832, -9.7886323928833, -2.4579079151153564, -6.094926834106445, -0.874626100063324, -6.043811321258545, 5.828069686889648, 0.9103712439537048, -6.006288051605225, 1.6235713958740234], [-1.6554802656173706, -5.915804862976074, 1.5740050077438354, -3.851266860961914, -5.967731475830078, 5.491958141326904, 3.2613399028778076, 4.301260948181152, -3.328975200653076, 3.772900342941284, -1.3792239427566528, -2.4291882514953613, -11.826593399047852, 2.214240312576294, 5.332174777984619, 6.2262043952941895, -7.3724684715271, -10.560189247131348, -5.614070892333984, -8.213237762451172, 14.464051246643066, -3.8132271766662598, 2.8684258460998535, 0.49945592880249023, 0.352753609418869, -10.32015609741211, -0.6259657740592957, -14.776644706726074, -0.7845008969306946, -2.470750331878662, 2.056225299835205, -9.044500350952148, 0.47851550579071045, 3.109809398651123, -3.893291473388672, 2.9691765308380127, 0.17820900678634644, 0.811698317527771, -5.317644119262695, 5.852074146270752], [11.433910369873047, 7.456541061401367, 6.59627628326416, -2.4698092937469482, -1.7294299602508545, -3.6028172969818115, 8.538105010986328, -1.2607694864273071, -1.3501904010772705, 5.582197189331055, 11.756731986999512, -3.5547635555267334, -5.6917524337768555, -1.6687185764312744, 3.6333296298980713, -9.975479125976562, 2.7416653633117676, -0.6588984727859497, 5.352788925170898, -2.0030698776245117, -5.281811237335205, -6.712594032287598, -5.0545830726623535, -9.579703330993652, -4.680801868438721, 1.0346676111221313, -5.7221269607543945, -1.792266845703125, -3.705893039703369, -6.89309549331665, 2.691988706588745, -0.5060181021690369, 3.3592677116394043, -1.8951096534729004, -1.8294191360473633, -1.5671205520629883, 5.46719217300415, -10.135499954223633, -0.47082144021987915, -8.293880462646484], [-6.960679054260254, -8.029998779296875, 11.138967514038086, -8.634115219116211, 8.555891036987305, 5.874197959899902, -12.68932056427002, 5.892143249511719, -9.203131675720215, 15.902735710144043, -2.486440896987915, 1.320628046989441, 13.018145561218262, -1.713663101196289, -3.547081708908081, -4.224106311798096, 4.643220901489258, 7.640494346618652, -17.385967254638672, 3.5562195777893066, -0.638797402381897, 9.931953430175781, -6.493816375732422, 7.027276039123535, 4.957559585571289, -0.7681450247764587, 5.678891658782959, -5.0219807624816895, -8.463014602661133, -0.1372661590576172, 2.88198184967041, -0.03144615888595581, -7.747935771942139, 0.3237341344356537, 3.910539388656616, 6.962835788726807, 4.063976287841797, 16.59604835510254, -3.977108955383301, -2.735992908477783], [-2.209104061126709, 4.821651458740234, -3.3593313694000244, -4.76235294342041, -6.9989728927612305, -6.241683006286621, -12.427096366882324, -12.334425926208496, -4.017565727233887, 6.398318767547607, 1.7585480213165283, 6.34575080871582, -6.182343006134033, -3.8488006591796875, 6.486279010772705, -3.5654356479644775, -2.8333182334899902, 13.155635833740234, 8.292048454284668, -8.858970642089844, 4.895260334014893, 0.4221324026584625, 1.0809170007705688, -1.2283378839492798, 6.754183769226074, 5.57023811340332, 8.88968563079834, -9.831787109375, 1.67118239402771, 7.3761773109436035, 6.547577381134033, -5.602548122406006, 4.767553329467773, 6.145163536071777, 2.1634514331817627, -3.8868324756622314, -3.199669361114502, 3.3725383281707764, 0.5216418504714966, -4.264280319213867], [2.0577192306518555, -0.48897451162338257, -7.853557586669922, -4.385944366455078, -1.8719894886016846, 2.5631518363952637, 5.591521739959717, -8.969508171081543, 0.4837362766265869, 2.103480100631714, 3.351947546005249, -5.417960166931152, -9.266767501831055, 9.083803176879883, 3.6624362468719482, -6.851893424987793, 9.963604927062988, 1.08319091796875, 1.3263803720474243, -8.66350269317627, 0.26425132155418396, -2.8484628200531006, 7.41701078414917, 0.5494946837425232, -2.3065414428710938, 3.1998040676116943, -9.243566513061523, -0.16841939091682434, -1.4259886741638184, -11.642518997192383, -0.6681144833564758, -3.4300050735473633, -10.111438751220703, 2.579786777496338, 2.6805660724639893, 6.3133649826049805, -1.5155917406082153, -6.211615085601807, 0.018345428630709648, 3.802656650543213], [0.8686649799346924, -8.285896301269531, 2.0408408641815186, -8.356998443603516, 4.628677845001221, 1.7339630126953125, -1.9668782949447632, 1.9270774126052856, 0.018025854602456093, 16.37538719177246, -3.072906494140625, 5.108872890472412, -3.0724170207977295, -8.252227783203125, 2.8856396675109863, 3.024418830871582, 1.5775121450424194, 1.4247530698776245, 0.4089304506778717, 1.0124379396438599, -6.6510419845581055, -4.570311069488525, -3.875000238418579, -13.187289237976074, 6.00347375869751, 5.959733963012695, 1.988892912864685, 6.215967655181885, 5.970495223999023, -0.027227230370044708, -3.7545976638793945, 2.1357944011688232, 2.7199795246124268, 0.7457693815231323, 6.675289630889893, 0.361236572265625, -0.08681570738554001, -3.058471918106079, 6.099185466766357, -15.512628555297852], [1.8890306949615479, -1.9869121313095093, -8.149266242980957, 8.025530815124512, -1.063881754875183, 2.9077723026275635, -9.610885620117188, -8.129889488220215, -6.382358551025391, 2.3945188522338867, -4.113139629364014, -0.17344845831394196, 18.545272827148438, -6.807958602905273, -0.476468563079834, 5.252971172332764, 9.817573547363281, 8.854334831237793, -7.182948112487793, 1.851302981376648, -2.376168966293335, -2.547968626022339, -5.976525783538818, -2.9579415321350098, -8.975322723388672, 5.9423370361328125, 4.236293315887451, 0.09569764882326126, 2.9913525581359863, 0.7442767024040222, -2.561659097671509, 3.2581915855407715, -4.345914363861084, 9.525618553161621, 0.7233747839927673, -8.209261894226074, -10.187335968017578, -7.01315450668335, 3.86126971244812, 1.8452565670013428], [2.9241631031036377, -9.79191780090332, 3.711026668548584, 11.051237106323242, 5.077682971954346, 7.068098068237305, 6.284094333648682, 4.1645331382751465, 4.4084882736206055, -2.346569061279297, 7.349018096923828, -2.0160350799560547, -8.71616268157959, -3.579078435897827, 6.581342697143555, -2.8577754497528076, 8.781489372253418, 0.3355461657047272, 9.581487655639648, -4.7484869956970215, 7.446226119995117, 4.112372875213623, 3.3139028549194336, 11.087312698364258, -10.447637557983398, -8.967037200927734, 5.553752899169922, -2.220407485961914, 0.6797070503234863, 3.1116974353790283, 5.222310543060303, -2.563667058944702, 5.740298748016357, 11.94339656829834, 0.9155121445655823, -5.286036014556885, -2.8567869663238525, -6.6966681480407715, -4.721148490905762, 3.8105223178863525], [-7.102904319763184, -3.9729208946228027, -6.8010406494140625, -10.575480461120605, -2.7479608058929443, -6.040053844451904, -8.34133243560791, -7.9438557624816895, -8.182844161987305, 9.695785522460938, -0.2576232850551605, -3.896908760070801, -0.31841716170310974, -2.5831196308135986, 6.7358598709106445, 8.65993595123291, 6.051366806030273, 3.830632448196411, 11.85978889465332, -9.421260833740234, -1.9954049587249756, 1.5314040184020996, 0.5587499141693115, 4.943766117095947, -4.038987159729004, 8.385727882385254, -12.432891845703125, 2.9477813243865967, 3.605599880218506, -5.263144493103027, 11.779158592224121, 9.302216529846191, 4.3061065673828125, 5.314996242523193, 0.013405006378889084, 5.1059441566467285, -6.5773210525512695, -3.3040261268615723, -7.255589485168457, -4.71345853805542], [0.6305853128433228, -5.546504497528076, -5.376166820526123, -4.7163615226745605, -2.6887831687927246, 11.83155345916748, 2.106973886489868, 9.04605770111084, 10.181565284729004, 2.7238714694976807, -3.2812042236328125, 1.5822126865386963, 2.240234613418579, -3.4210779666900635, -4.737635135650635, -1.2852756977081299, 3.7031400203704834, -8.981881141662598, -2.429619073867798, -1.4424927234649658, 6.584481716156006, 5.492712497711182, -0.4077484905719757, 4.485854625701904, 0.1523890495300293, -2.4851741790771484, 1.5087263584136963, -8.123059272766113, 4.506482124328613, 4.565250396728516, -5.9871344566345215, 1.6788904666900635, -7.564218044281006, 0.963472306728363, -5.449982166290283, 2.484813928604126, 1.29178786277771, 1.3652523756027222, 2.8084516525268555, 3.9827659130096436]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.0118653774261475, -0.350004643201828, 0.9476233124732971, -0.4152289927005768, -0.04665598273277283, -0.914202094078064, -0.2873157560825348, -0.717067539691925, 1.3248940706253052, 0.13137374818325043, -0.42209964990615845, -1.409089207649231, 1.2170405387878418, 0.44218695163726807, -0.21630699932575226, -0.310416042804718, 0.8819696307182312]], "dtype": "float32"}, {"__nd__": [[-0.2036607414484024], [0.46233320236206055], [-1.4534649848937988], [-1.5681231021881104], [1.168402910232544], [-0.9198525547981262], [-0.34772759675979614], [0.44957685470581055], [-1.1141151189804077], [-0.7190955877304077], [-0.005803947802633047], [0.5677075386047363], [-0.09777259826660156], [-2.084679365158081], [1.1269116401672363], [0.13598792254924774], [-0.7380854487419128]], "dtype": "float32"}, 5, 16], "e": {"__tup__": [{"__nd__": [[-4.463807106018066]], "dtype": "float32"}, {"__nd__": [[[-3.812838315963745]], [[-0.6509689688682556]], [[0.0]], [[0.0]], [[0.0]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.7114595770835876, -0.2821996212005615, -0.5995240211486816, 0.2906639277935028, 0.20150186121463776, -1.2418363094329834, -0.875783383846283, -1.299842119216919, 0.8575913906097412, 0.7477796673774719, -1.3070491552352905, 0.2082972377538681, 2.052175283432007, 0.3259184956550598, -1.0570261478424072, 1.7494349479675293, 0.1806603968143463, 0.5933748483657837, -0.42761918902397156, -0.6960088014602661, 0.7839271426200867, 2.688885450363159, -0.16266435384750366, 1.3328698873519897, 0.8749774098396301, -0.3585354685783386, 0.9161961078643799, 0.13147161900997162, -0.0631885975599289, -0.4852534830570221, 0.6567534804344177, -1.3944816589355469, -1.0537593364715576, -0.08901916444301605, 0.6837518811225891, -0.16171357035636902, 0.3985500633716583, 0.27190685272216797, -0.47656548023223877, 1.0117859840393066, -1.0684130191802979, 0.3365292251110077, 0.09484551846981049, 1.29300856590271, 0.8118860125541687, -0.9740120768547058, 0.7195126414299011, -0.9070031046867371, 0.6398596167564392, -0.46025633811950684, 0.3058561384677887, -0.07937110960483551, -0.846646785736084, 1.0833760499954224, 0.5182068347930908, -0.05735187605023384, -0.13277079164981842, 0.8512808680534363, -0.4379979372024536, -0.8907281160354614, 0.9668447971343994, 0.3955744206905365, -0.9095038771629333, 0.02181309275329113, 1.1646541357040405, 1.3464874029159546, 1.292513132095337, -1.7252404689788818, 0.38341113924980164, 0.0756072849035263, 0.11303136497735977, -1.0321136713027954, 0.9710531234741211, 0.19958321750164032, -0.8923774361610413, 0.2522713541984558, 1.1606663465499878, 0.65955650806427, -0.6180132627487183, -0.7842941880226135, 0.45618128776550293, -0.6433801651000977, -1.231968641281128, 1.669960856437683, -1.0727264881134033, 0.48068907856941223, -1.2551631927490234, 1.7515877485275269, -0.5018689632415771, -0.3248472213745117, -0.8657257556915283, 0.13614007830619812, 2.1920430660247803, 0.2347310334444046, -1.5777391195297241, 0.6105480790138245, 0.4950854778289795, -1.7973405122756958, -0.06539550423622131, -1.1837632656097412, -1.4061856269836426, 0.9743812084197998, -0.044622618705034256, 0.8170031309127808, -0.17149969935417175, -0.4755432605743408, -1.0969616174697876, -0.2596684992313385, -0.4073165953159332, 0.8711516857147217, 0.7504432797431946, -1.1868515014648438, -0.4766486585140228, 1.7736769914627075, -0.578811526298523, 0.2471303641796112, -1.2768440246582031, 0.6913011074066162, -0.7011746764183044, 0.3684074878692627, -0.9950222969055176, 1.33370041847229, -0.19763857126235962, 0.06911933422088623, -0.6939971446990967, -1.416290044784546, 0.7710981369018555, 2.105959177017212, 0.8957254886627197], [0.2643669843673706, 1.631143569946289, 0.6634269952774048, 1.1121032238006592, 1.4192981719970703, -0.9083177447319031, 0.12883034348487854, 0.6893475651741028, 0.45497941970825195, -0.8904709815979004, -0.08407878130674362, -0.34975868463516235, 1.4888641834259033, 0.24858392775058746, 0.2650281488895416, 2.092517852783203, -0.8581123948097229, 0.7692722082138062, -0.693401575088501, 0.11331408470869064, -0.3540603518486023, 0.7539380788803101, -1.1486519575119019, 0.18072842061519623, -0.34357142448425293, -0.885645866394043, -1.5020941495895386, -0.9787224531173706, 2.6332085132598877, 0.1477326601743698, -1.0227534770965576, -0.959599494934082, 1.7450367212295532, 1.6022770404815674, 1.7082284688949585, -0.38651198148727417, -0.02484431117773056, 1.3240388631820679, -0.5833616256713867, 0.12062728404998779, 0.20811745524406433, -0.7083004117012024, -1.0960389375686646, -0.5702927708625793, -0.049871183931827545, 2.0889225006103516, -0.18530738353729248, -0.45417505502700806, -0.5006305575370789, -0.2382218837738037, -1.000303030014038, 1.6389095783233643, -2.032935380935669, -0.89960777759552, -0.4111255705356598, -0.7750615477561951, 0.33648744225502014, 1.3544719219207764, -1.1565862894058228, 0.6268084645271301, 0.6841628551483154, 0.9209256172180176, -0.7164857983589172, -2.9288694858551025, 0.8419629335403442, 1.4438556432724, -0.4926150441169739, 0.5193057060241699, -0.5167374014854431, -0.4913480281829834, 0.11315947026014328, 1.5950757265090942, -1.9769290685653687, 0.02699480764567852, -1.6101998090744019, 0.5141541957855225, 1.2778996229171753, 0.6065434813499451, -1.646748661994934, 0.036988258361816406, 1.0985291004180908, -0.9069379568099976, -0.1252797245979309, -0.3971070349216461, 0.46879130601882935, -0.06082834675908089, 1.0039821863174438, -1.3770605325698853, -0.6813081502914429, 0.005860087461769581, -0.3895953297615051, -0.9498875737190247, -0.8280229568481445, 1.375044822692871, -0.24741171300411224, -1.3319884538650513, 0.3088499903678894, 0.10164980590343475, -1.957027554512024, 0.4581015110015869, -1.4156761169433594, 1.4603731632232666, 1.4910835027694702, -0.5049618482589722, -0.30076122283935547, -1.396523356437683, 0.007361788302659988, 0.6832754611968994, 0.5365022420883179, 0.5446399450302124, -1.5895763635635376, 1.3034559488296509, -0.4842473268508911, 1.5810742378234863, -1.6941637992858887, 1.3480279445648193, 0.5490100383758545, -0.3072943687438965, 0.31789112091064453, 1.2026069164276123, -0.14159977436065674, 0.5732941031455994, -0.03498360514640808, 1.2526177167892456, -0.4820006191730499, -2.3863282203674316, -1.0750547647476196, 0.6560852527618408, 1.3397538661956787], [-1.1103434562683105, -1.4899381399154663, -1.1245893239974976, -0.09658132493495941, 0.6671741008758545, 0.415786474943161, 1.620347499847412, -0.9097033143043518, 1.3025610446929932, -1.391695499420166, 1.0522429943084717, -0.8571982383728027, -0.5986818671226501, -0.59744793176651, -0.5063878297805786, -0.3666714131832123, -0.8056901693344116, -0.8020090460777283, 0.41008901596069336, 0.6488732695579529, -0.27547428011894226, -2.077097177505493, 0.04602714627981186, -0.3421866297721863, 1.1237847805023193, 1.3146321773529053, 0.8939698934555054, 0.6423734426498413, -0.7641934752464294, -2.6485142707824707, 0.016943221911787987, 0.9655954837799072, 0.42424044013023376, 0.41702553629875183, 0.7927991151809692, 0.8322360515594482, -0.1754046082496643, 1.3434131145477295, 0.01891307160258293, 2.0475528240203857, -2.0056183338165283, 0.9518441557884216, 1.1518807411193848, -2.8961901664733887, 0.011254467070102692, -1.6103298664093018, -0.13021549582481384, -2.5539448261260986, 1.218800663948059, -0.4557077884674072, 0.4195021986961365, -0.05925661697983742, 2.7361881732940674, -0.84267657995224, 0.6768515110015869, -1.4898074865341187, -0.22747676074504852, -0.2999268174171448, -0.09058072417974472, -0.8648965358734131, 1.400084137916565, 1.4142266511917114, 1.5579200983047485, -0.5585614442825317, 0.1540694385766983, 1.1081818342208862, -0.5342866778373718, 0.505842387676239, 0.5496299862861633, -0.8638942837715149, 0.3008131980895996, 0.45028823614120483, 0.5895715355873108, 0.2340007722377777, 1.3175227642059326, 1.111889123916626, 0.7856327295303345, 0.08912026137113571, -1.7219480276107788, -0.4378076195716858, -1.3187799453735352, -2.9319863319396973, 0.1494242250919342, -0.3765462040901184, 2.1005327701568604, -0.4895804226398468, 1.5505272150039673, -1.8754420280456543, -0.2595180571079254, -2.275998830795288, 0.716396689414978, 0.006923098582774401, -0.2415560632944107, 0.7114918828010559, 1.2707891464233398, -0.7322922945022583, -0.3147980272769928, 0.9212279319763184, -0.03158659115433693, 0.6973443627357483, 0.8558599352836609, 0.641822099685669, -1.5850448608398438, -0.17080873250961304, 0.0884118303656578, 1.1552926301956177, 0.5594409704208374, -0.6318402290344238, 0.2746736407279968, -1.354982852935791, 0.4419889450073242, 0.7969435453414917, -0.5269152522087097, -0.438469797372818, -0.2667864263057709, 1.1090558767318726, -1.7634506225585938, 0.06192280724644661, -0.1840398758649826, -0.44483810663223267, 0.8219484686851501, 0.10942712426185608, -0.7602264285087585, 1.2256540060043335, 0.4866727888584137, -0.03959566354751587, 1.3521630764007568, 1.1828011274337769, 0.007709316909313202], [0.30931612849235535, -0.9580411314964294, 0.3086615204811096, -0.94264155626297, 0.45835351943969727, 0.6098924875259399, 0.0481230765581131, -1.0716993808746338, 0.9995262622833252, 0.8920784592628479, -0.1674933135509491, -0.6020315289497375, -0.3276121914386749, 0.394800066947937, 0.38282904028892517, 0.9563212394714355, -0.4827336072921753, -0.4898103177547455, 0.3984297215938568, 0.3820253312587738, -0.5286110639572144, -0.49561014771461487, 0.04382002726197243, 2.156890630722046, 0.9981695413589478, -0.02466988004744053, 0.6769322156906128, -0.856514036655426, -0.7407065033912659, 0.63117915391922, -0.4237704575061798, 1.3084288835525513, 0.8931440114974976, 0.1093912199139595, 0.6437100172042847, 0.6507951021194458, 0.6182585954666138, 1.2071295976638794, -0.4034731090068817, 0.08624476194381714, -0.5938034653663635, 0.6451467275619507, -0.0827423706650734, -0.40716683864593506, -0.31308841705322266, -1.5732700824737549, -2.6746277809143066, -1.4330120086669922, -1.1981191635131836, -0.08819416910409927, -0.9752411246299744, 1.3052743673324585, -0.20254741609096527, 0.7748925089836121, 0.697355329990387, -1.3107696771621704, -0.6004555225372314, -1.8100420236587524, 0.5050013065338135, 1.6179425716400146, -0.17541249096393585, -0.020070798695087433, 0.6109297275543213, 0.38607895374298096, 1.1867928504943848, -0.19252970814704895, 1.2381923198699951, -0.23317982256412506, 2.0094401836395264, -0.5601657629013062, 0.448757141828537, 0.249151349067688, -0.09562861174345016, -0.6269721388816833, -0.6298030018806458, -0.450147807598114, -0.42921558022499084, -1.3745061159133911, -0.003507832298055291, -1.147230863571167, 1.2108242511749268, 0.34258729219436646, -0.8747783303260803, -0.3887690007686615, 0.4960794150829315, -1.0689306259155273, 0.512355625629425, 1.0304596424102783, 0.3571849763393402, -0.15835972130298615, 0.6128121018409729, -0.9818267822265625, -0.3935343325138092, -0.697950005531311, 1.838947057723999, 0.5436572432518005, -0.8847874999046326, 0.47641366720199585, 0.07903233170509338, 2.996633291244507, -0.5055091977119446, -0.11561573296785355, 0.4913712739944458, -1.217800259590149, -0.7815567255020142, -1.0694785118103027, 0.7232498526573181, -1.4731420278549194, -0.801207423210144, 1.4719914197921753, 0.7060911059379578, 1.3491629362106323, -1.0097217559814453, -0.6557814478874207, 0.680810809135437, 0.20536233484745026, 0.3656068742275238, 0.2248205691576004, 0.29025593400001526, 1.0778043270111084, 0.2869364023208618, -0.5113014578819275, 1.0136988162994385, 1.228410243988037, -1.006855845451355, 0.9028195738792419, 1.0595253705978394, -0.16739003360271454, -1.3048675060272217], [-2.2126545906066895, -0.3351558446884155, 0.45160186290740967, 0.3855622708797455, -1.030150055885315, -0.9296605587005615, 1.4433908462524414, -1.4342454671859741, -0.11776236444711685, 0.33345308899879456, 0.5322259664535522, -0.45730310678482056, 1.029871940612793, 0.1576581746339798, 0.20168477296829224, 0.08991033583879471, -1.731089472770691, 1.6528756618499756, 0.36842504143714905, -0.38215917348861694, -1.0815037488937378, 0.0051239291206002235, 0.8403487205505371, -0.1871718168258667, -0.17725272476673126, -1.751842975616455, -0.48519888520240784, 0.18636412918567657, 2.199465274810791, -1.1697901487350464, 1.5961228609085083, 0.9532643556594849, -1.2358187437057495, -0.42399194836616516, 0.8785860538482666, 2.7741754055023193, 1.0055612325668335, 1.1070077419281006, 0.16065596044063568, -0.7126712799072266, 0.3417637348175049, 0.2640978693962097, 0.03440501168370247, 2.5293073654174805, -2.3134851455688477, -2.098031520843506, 1.260011911392212, 1.9394259452819824, 0.8872995972633362, 0.3886057138442993, -1.1904557943344116, -0.8238610029220581, 0.08691941201686859, 2.2048816680908203, 1.5335612297058105, 0.49464747309684753, 0.7403264045715332, -0.1930103451013565, -0.3835555613040924, 1.2526816129684448, -0.19742321968078613, 1.022124171257019, -0.571435272693634, -0.5388372540473938, 0.11578654497861862, 0.48331698775291443, 0.2121104896068573, -0.23925888538360596, -1.94497549533844, 1.4305915832519531, 1.2738964557647705, 0.4251249134540558, 0.11704158782958984, 1.1811237335205078, -0.5109149813652039, -1.29501473903656, 0.4258202612400055, -0.41864708065986633, -0.8117780685424805, 0.9649763703346252, 0.32451528310775757, 1.1208035945892334, 1.0864136219024658, 1.4886468648910522, -0.5554342865943909, 0.611569881439209, 1.356019377708435, -0.04697369411587715, -0.009770520962774754, 0.891619086265564, 0.17347034811973572, 0.6102007627487183, 0.3614535629749298, 0.12413299083709717, -0.46988075971603394, -1.112971544265747, 1.1608587503433228, -0.6406819224357605, 0.3363250494003296, 0.871638834476471, 0.7246531248092651, 0.5786157846450806, 0.937342643737793, -0.2513628304004669, 0.8396142721176147, 1.0618846416473389, 1.448418378829956, 0.1761537492275238, 0.18673880398273468, -0.5750186443328857, -0.475098580121994, -1.0819555521011353, -0.5760607719421387, -2.0511138439178467, 0.060514286160469055, 2.3975133895874023, -0.6280057430267334, -1.5357941389083862, -0.06936806440353394, 0.1285131871700287, 1.0919502973556519, -0.9837520718574524, 0.018252799287438393, 0.15154963731765747, 1.8717505931854248, 0.9325895309448242, -1.1398289203643799, 0.44863444566726685, 0.45113232731819153], [-2.490797996520996, -1.6029108762741089, -1.5970759391784668, -0.3318697214126587, 1.7291265726089478, -1.3828885555267334, -0.32435527443885803, 0.05406685918569565, -0.20309267938137054, -1.7672734260559082, -0.40707382559776306, -0.5264357328414917, 0.6109784841537476, 0.7290382385253906, 2.373567581176758, -0.904378354549408, 1.5971651077270508, 0.9066818356513977, -0.27757471799850464, -1.4534631967544556, 0.16333399713039398, -0.07277506589889526, 0.8978315591812134, -0.5555537939071655, -0.010996400378644466, 0.19195102155208588, 0.13585171103477478, 0.9475474953651428, 0.6564396023750305, 0.8380804061889648, 0.6618403792381287, -1.828210473060608, 0.20571815967559814, -0.2077770233154297, 0.4480400085449219, -0.9844995737075806, 0.8657417297363281, 0.6747908592224121, 0.28021469712257385, 0.3120698928833008, -0.5879397988319397, 1.0313241481781006, 1.2356481552124023, 1.2100324630737305, -2.6300618648529053, 0.6980156302452087, -0.6026734113693237, 1.0460076332092285, 0.46906331181526184, 2.200451374053955, -1.6908843517303467, -2.101618766784668, -1.0542455911636353, -0.37497860193252563, -0.1797916442155838, 0.5007139444351196, 1.185409426689148, -1.1854132413864136, -0.3524135649204254, -1.0639458894729614, -1.3645800352096558, -0.13476504385471344, -0.7811278104782104, -1.4518630504608154, -0.4486246407032013, 0.881210207939148, 0.3675673305988312, -0.21910864114761353, -1.3164948225021362, 0.6644209623336792, 0.0010744633618742228, -1.2976419925689697, -0.21408987045288086, -0.50872802734375, -0.3442610502243042, -0.3404800593852997, 1.2102199792861938, -0.3108230233192444, -0.2804704010486603, 1.2010213136672974, 1.261717677116394, 1.302308440208435, 1.1893658638000488, -2.115699291229248, 1.1881996393203735, -0.3488698899745941, 0.20901638269424438, 1.4883934259414673, 0.9822184443473816, -1.7569679021835327, 2.6623318195343018, 0.2168930470943451, -0.6911017894744873, 0.307471364736557, 1.5680986642837524, -0.6283766627311707, -0.2101055085659027, -0.817805290222168, 0.2946438491344452, -0.5991492867469788, 0.05962642282247543, 1.3976867198944092, -0.6346569061279297, -2.4232892990112305, 2.339823007583618, 0.4083789885044098, -1.7752934694290161, 0.8783257603645325, -0.8070162534713745, 0.8139955401420593, -0.7360823750495911, 1.3194169998168945, -1.0297073125839233, -0.5649110674858093, 0.21050825715065002, 1.7572407722473145, 0.07341368496417999, -0.03803978115320206, -1.1165484189987183, -0.4343743920326233, 0.6060704588890076, 0.6593454480171204, -0.35388267040252686, -1.0733413696289062, -0.03230457380414009, -0.8035188913345337, -1.0780888795852661, -0.4338664710521698, 0.012401890940964222], [1.6702549457550049, -0.11246529966592789, -1.958815336227417, 1.2694458961486816, -0.3689034581184387, 1.348698377609253, -1.49779212474823, 0.3922441601753235, 0.10957878082990646, -2.3694963455200195, -0.1728411763906479, 0.4568326473236084, -1.3681026697158813, -1.6474336385726929, 0.1351306289434433, 0.86016446352005, 0.8944949507713318, 1.4112918376922607, -0.28860893845558167, 0.561567485332489, 0.6323409676551819, 0.26002851128578186, -0.09600566327571869, -0.46209031343460083, 1.152638554573059, -2.1351499557495117, -1.1953790187835693, 0.10382059216499329, -0.12179072201251984, -0.8375192880630493, 2.158898115158081, 0.01477972324937582, -2.133578062057495, 1.7225265502929688, 0.605656087398529, 2.131533622741699, 0.3478594422340393, -0.34493115544319153, 0.2970051169395447, 0.3726469874382019, -0.03512796759605408, 0.4035032391548157, 0.1487911194562912, -0.2146425098180771, -0.5658740997314453, 0.13625428080558777, -0.9641516208648682, 0.18991613388061523, 0.36035749316215515, -0.824690043926239, -2.330298900604248, -0.7203889489173889, -0.7407771944999695, -0.575239896774292, 0.4800323247909546, -0.12111804634332657, -0.4064662456512451, 0.014651716686785221, -0.17578963935375214, -1.3059055805206299, -1.4207693338394165, 1.1169028282165527, 0.5192465782165527, -0.7730809450149536, 0.8505051136016846, -0.7988926768302917, -1.001945972442627, 0.6422332525253296, 0.2530606985092163, -1.8165074586868286, -0.2803684175014496, 0.0860501378774643, 1.1078999042510986, -0.997648298740387, 0.5188589692115784, 0.053072601556777954, -1.1142160892486572, -1.569203495979309, 2.1442744731903076, 0.47126007080078125, -1.7614272832870483, -0.8331933617591858, 1.4266120195388794, 0.31145018339157104, 0.157329723238945, 1.4157447814941406, 0.9410626292228699, 0.4236123263835907, 1.3531869649887085, -0.37288540601730347, -1.2144466638565063, -1.135515570640564, 1.117444634437561, -1.6303908824920654, 2.225412130355835, 0.27630358934402466, 0.25706082582473755, 0.5418344736099243, -0.7080204486846924, -1.1899958848953247, 1.117065668106079, -1.2347774505615234, -0.42675694823265076, 1.4044898748397827, 0.36530375480651855, 1.3322291374206543, 1.7361140251159668, 0.030521340668201447, 0.6025563478469849, -1.0640310049057007, 2.2463343143463135, -3.4011495113372803, 1.1163551807403564, -0.16305263340473175, 0.4626903831958771, 0.5642236471176147, 0.9451044201850891, -0.8160795569419861, -0.23852579295635223, -0.3311534821987152, -1.1229496002197266, -0.5413122177124023, -1.3461092710494995, -0.21515288949012756, 1.10076105594635, -0.2081844061613083, 0.25276824831962585, -1.9701601266860962, -0.10059741884469986], [1.3613760471343994, 1.5200632810592651, 0.25451943278312683, -1.4058325290679932, -0.7574931383132935, 1.3399409055709839, 0.4730335474014282, -0.4833020865917206, -0.6745972633361816, 1.0211262702941895, -1.0598294734954834, 0.3564613163471222, -0.7643166780471802, -0.31111228466033936, 0.928870439529419, -0.5807100534439087, 0.9861080646514893, -0.03990627080202103, -0.593658447265625, 0.3453614413738251, 0.5233325362205505, 1.1485657691955566, -1.0781843662261963, 0.6919119358062744, 0.62008136510849, -0.6770610213279724, 1.5487072467803955, 1.2419081926345825, -0.9475817680358887, 0.9638970494270325, -0.6894943118095398, 0.31996601819992065, -1.7351884841918945, -0.15237216651439667, 0.16782978177070618, 0.03523050621151924, -0.3418099284172058, -1.220990777015686, 1.4268726110458374, 0.5347163677215576, 0.2385983169078827, 1.5523154735565186, 1.3232344388961792, -0.728458046913147, 1.495147943496704, -0.7910993099212646, -0.39806869626045227, -0.3821238875389099, 0.3676773011684418, 1.2436842918395996, -0.1839730143547058, 0.6660984754562378, -0.1940459907054901, -0.938564658164978, -1.0434027910232544, 1.6621215343475342, -0.6373088955879211, 0.32038357853889465, 0.9364941120147705, 0.07730576395988464, -0.1613152176141739, 1.1524484157562256, 0.23193509876728058, -0.0888671949505806, -1.2370294332504272, -0.2589483857154846, 0.7658812999725342, 0.19171585142612457, 0.8547718524932861, -1.405661940574646, -0.6790273189544678, -0.9126566648483276, -0.6773661375045776, -0.5907169580459595, 0.0035443666856735945, 1.0925580263137817, 0.4156816601753235, 0.9530454874038696, -1.5956852436065674, -0.517436146736145, 1.6003403663635254, -0.21207879483699799, 0.42332732677459717, -0.4630473852157593, 0.2028946727514267, -0.600407063961029, 1.546452522277832, -0.0844128206372261, 0.2975352704524994, -0.493975967168808, 0.5395107865333557, 0.52317214012146, 0.21736998856067657, -0.016773201525211334, 0.5174551010131836, -1.6722081899642944, 1.5260441303253174, -1.884352684020996, -0.4342538118362427, -0.8965680599212646, 1.201881766319275, 0.6865620017051697, 2.3112635612487793, 0.6313633322715759, 0.9939296245574951, -0.503652811050415, -0.8153038620948792, -1.179431438446045, -2.034592390060425, 1.177608609199524, -0.44745108485221863, 1.0917414426803589, -0.868728518486023, -1.030423641204834, 1.3240407705307007, 0.1785302460193634, -0.266834020614624, -0.36766305565834045, -0.13921630382537842, 0.9734644293785095, -0.058806173503398895, 0.8246877193450928, -1.5309101343154907, 0.7678738832473755, -1.123927116394043, 0.12420357018709183, -0.16566573083400726, -0.18163315951824188, 1.050837516784668], [1.4116805791854858, -1.3268193006515503, 1.2950221300125122, -2.214696168899536, -0.7123767733573914, 1.5738276243209839, 0.5478841066360474, -0.26076608896255493, 0.2815864384174347, 0.3914926052093506, -0.418373167514801, -0.2657005190849304, 0.3287581205368042, -1.907052993774414, -1.0606317520141602, -0.6793845891952515, 0.30327141284942627, -0.7931854128837585, 0.9868959784507751, 0.17867344617843628, 0.964165449142456, 1.1594754457473755, 0.5673783421516418, -0.4724251925945282, 0.6468744874000549, -0.6992182731628418, 0.8886882662773132, 1.3033702373504639, 0.4008397161960602, 0.2170703411102295, 1.4017384052276611, 1.2667690515518188, 2.64823317527771, -1.4292865991592407, -1.6957250833511353, -0.04533300921320915, 0.008915980346500874, 0.06284547597169876, 0.2352217137813568, 0.10129919648170471, 0.17509473860263824, -0.5783966183662415, -1.1409826278686523, 0.8452557921409607, 0.6957387924194336, -0.3536188304424286, -0.9111163020133972, 0.5581983327865601, -1.3074063062667847, -1.6542835235595703, -0.13899041712284088, -0.7433274388313293, 0.02422316186130047, 0.8201315402984619, -0.06639132648706436, 1.0946614742279053, -1.1359047889709473, 1.5542984008789062, 0.6486527919769287, 1.541993498802185, -0.5338135957717896, 1.8083714246749878, -0.3466106355190277, 0.2511700987815857, -1.1417272090911865, 0.892928421497345, 1.3906482458114624, -0.7696327567100525, -0.08347701281309128, 0.492958128452301, 1.2733041048049927, -1.4116290807724, 1.6935052871704102, -1.2604140043258667, 0.6681958436965942, -0.9610229134559631, -0.26404350996017456, 0.12552762031555176, 1.348311185836792, 0.16739828884601593, 2.153839588165283, 0.6355680823326111, -0.6871585249900818, -1.6549835205078125, -1.1880789995193481, -1.4845855236053467, -0.6214724779129028, 0.5683683156967163, 1.0624606609344482, -0.4600822627544403, 0.20866161584854126, 0.9956633448600769, 0.7714459896087646, -1.3242793083190918, 1.7639415264129639, 0.7501552700996399, 0.8347111940383911, 0.34399592876434326, 0.5020313262939453, -0.8031884431838989, 0.12559360265731812, 0.478700190782547, -0.028749024495482445, -0.25396549701690674, -0.4739764928817749, 1.463456630706787, -2.443739175796509, -0.5337195992469788, -0.9321710467338562, -0.14628082513809204, -0.6823957562446594, 1.9237598180770874, 0.9739289879798889, -0.24231064319610596, 0.7153806090354919, 0.1070418655872345, 1.0740115642547607, 0.4288078844547272, 0.9529858827590942, -0.033325448632240295, -0.5972456336021423, 0.7579514980316162, 0.09092660248279572, 0.42633551359176636, -1.1383286714553833, -0.015092912130057812, 1.6235309839248657, 1.5218486785888672, 0.40435734391212463], [-1.8274834156036377, 0.9461204409599304, -1.3162733316421509, -0.10071466118097305, 0.9141017198562622, 1.6763083934783936, 0.29294463992118835, -0.4811556339263916, -0.797110915184021, -0.27603986859321594, 0.37101590633392334, -0.7089219689369202, 0.05347646400332451, -0.5225940942764282, 1.059639811515808, -1.6956291198730469, 0.579156756401062, -0.1130664050579071, 1.0687694549560547, -0.16517400741577148, -1.5384076833724976, -0.5691455602645874, -0.4588865041732788, 0.4841362535953522, 0.9565972089767456, -0.6565883159637451, 0.4168693721294403, 0.08196459710597992, -0.23699790239334106, 0.19080154597759247, -1.8026304244995117, 1.0427346229553223, 1.4838120937347412, 2.0718252658843994, -1.4416773319244385, -0.10505057871341705, 0.20040972530841827, 0.5366812348365784, 0.9993634223937988, 1.6388343572616577, -0.0857045128941536, 0.6435620188713074, 1.3288602828979492, 0.049288369715213776, 0.6115458607673645, 0.808728814125061, 1.1130125522613525, -0.6881363987922668, -1.0356125831604004, -0.9494489431381226, 0.5264132022857666, -0.11461595445871353, -0.7877184152603149, -0.8220288753509521, -0.8180887699127197, 0.8834679126739502, -1.2794201374053955, 1.1641921997070312, 0.2927834391593933, 1.4058266878128052, 0.3777327537536621, -1.5744556188583374, -1.5507179498672485, 0.19696471095085144, -0.08135764300823212, -0.6351044178009033, 1.0066418647766113, -0.18154843151569366, 1.3021055459976196, 2.5053043365478516, -2.8009984493255615, 0.9173324108123779, 0.40037667751312256, 2.2464802265167236, 0.6122143268585205, -0.14216409623622894, -0.9593520760536194, 0.14804299175739288, -0.5160936713218689, -0.12470020353794098, 1.487443447113037, -0.5253814458847046, -0.9447888731956482, -0.050448548048734665, -0.12087429314851761, 0.5872458219528198, -0.5934913754463196, -1.2938697338104248, 0.16784097254276276, 1.5938726663589478, -0.6351239085197449, 0.7215135097503662, 0.2927205264568329, 0.3459867238998413, 0.736474335193634, 0.5939299464225769, 0.5627845525741577, 1.2792129516601562, -0.9239100217819214, -1.578499436378479, 0.5902999043464661, 0.0350106842815876, 2.238276720046997, 0.3368974030017853, 0.6902381181716919, -0.3859215974807739, 0.18780508637428284, -0.3580896854400635, 0.5644487142562866, -0.3515298366546631, 1.3490502834320068, -0.3598245084285736, 1.4145073890686035, 0.44271203875541687, -0.3864806294441223, -0.707618236541748, -0.5944076776504517, 0.5041871070861816, 0.10507287830114365, -2.047140121459961, -0.8992822170257568, 1.5995324850082397, 0.24326379597187042, -0.14343395829200745, 2.2621407508850098, -1.476280927658081, -1.0013084411621094, -0.3340678811073303, -0.5453529953956604], [-0.14415307343006134, 1.2157541513442993, 3.056821584701538, -0.7162424921989441, -0.17585599422454834, 2.1442298889160156, -0.7638497352600098, 1.3394453525543213, -0.8058883547782898, -0.09433773159980774, -1.1520365476608276, -1.5353471040725708, -0.4445595145225525, 1.0885937213897705, 0.7085281014442444, 1.1777472496032715, -0.6031593084335327, -0.5811479687690735, 1.1117706298828125, 0.013839918188750744, -0.062301263213157654, -0.5143384337425232, 0.40787139534950256, 0.29459792375564575, 0.018574148416519165, 0.4869576692581177, 1.0536588430404663, 0.841618537902832, -1.364493727684021, -1.6258398294448853, 1.2289670705795288, 1.0267808437347412, 0.6163504123687744, -1.0237054824829102, 0.3257688581943512, 0.7116957306861877, -0.8267989754676819, -0.9106431603431702, 0.5134358406066895, 1.4173539876937866, -0.5558380484580994, -0.8487608432769775, 1.86749267578125, -0.5153674483299255, 0.18764694035053253, 2.7014760971069336, -0.5757462382316589, 1.027579665184021, -1.2912219762802124, -0.340284138917923, -0.294477254152298, 0.32156819105148315, -0.41879770159721375, -0.12710849940776825, 1.0137076377868652, -0.1630990207195282, 0.8339948058128357, 0.3664412796497345, 1.5902721881866455, 0.8885828852653503, -0.6832749843597412, -0.2267228066921234, -0.6234298944473267, 0.76146000623703, -1.1854742765426636, -1.0117459297180176, 1.269438624382019, 0.27521058917045593, -0.09815109521150589, -0.034246064722537994, -1.6178925037384033, 1.2590256929397583, 0.9242355227470398, -0.6035885810852051, -0.35064369440078735, 0.3763706684112549, 0.6502148509025574, 0.09480719268321991, 0.8432586193084717, 0.3285386562347412, -0.9806110262870789, 0.45336687564849854, -1.863290548324585, 1.8752351999282837, 1.2631030082702637, -0.6190334558486938, 0.6187103390693665, 0.3552645444869995, -0.21686376631259918, -0.4562974274158478, 1.3009473085403442, 0.20312021672725677, 0.15028049051761627, 1.7124208211898804, 0.16250447928905487, 0.24549411237239838, 1.1249840259552002, -0.05733446776866913, -0.05824246257543564, 0.012420674785971642, 1.2562576532363892, -2.3790650367736816, -1.4449183940887451, 0.19180956482887268, 0.25627273321151733, -0.7479118704795837, 1.7202351093292236, -2.553269386291504, 0.7841066718101501, -0.5434311032295227, 1.7036802768707275, -0.28422215580940247, 0.11512895673513412, -0.3933188319206238, 0.46631595492362976, 0.17244085669517517, 0.7938395142555237, -0.8483055830001831, 0.3062116503715515, -0.6658366322517395, -2.909174680709839, 1.0427347421646118, -0.07830245047807693, -1.002540946006775, -0.22767028212547302, 0.750339150428772, 1.8170270919799805, 0.9804506897926331, 0.5130552649497986], [0.7758153676986694, -0.04379308223724365, 0.5303775072097778, 0.31724119186401367, 1.4122308492660522, 0.061221055686473846, 0.2605249583721161, 0.47794750332832336, 0.2931946814060211, 0.7689275741577148, 1.832432508468628, 0.7416073679924011, -0.7239207029342651, -0.28413018584251404, -0.14582931995391846, 0.09427814185619354, -1.1407779455184937, 0.4097505807876587, 0.052155446261167526, 1.7729555368423462, 0.5348880887031555, 1.1878376007080078, 0.5577666163444519, 0.796062171459198, 1.0569944381713867, 0.40553465485572815, 0.14892996847629547, -0.13826122879981995, -0.1307792216539383, -0.9027155041694641, 0.8520759344100952, -0.07281942665576935, -1.2440398931503296, 0.3151721656322479, 0.34002429246902466, -0.7344699501991272, -1.386283040046692, -0.5826449990272522, -0.6390559673309326, 0.3935209810733795, -0.032995596528053284, 0.7625975012779236, -1.248207688331604, 1.6319994926452637, -0.1700131744146347, -1.031394124031067, 0.5163726210594177, -0.6411006450653076, 0.6252416372299194, 0.047972671687603, 0.5342226624488831, 0.9422318339347839, 0.4655342698097229, -0.7484047412872314, 0.026888588443398476, -0.7908826470375061, -0.06349943578243256, -1.5067684650421143, 2.8866095542907715, 0.3687008321285248, 0.8673763275146484, -0.8809975385665894, -0.7117537260055542, -0.2623814642429352, 0.12255634367465973, 0.1885615736246109, 0.1647336184978485, -1.0112496614456177, -0.7744640111923218, 0.8277532458305359, -1.269684910774231, 2.3549463748931885, 0.1793859601020813, -0.1381726711988449, 1.1841583251953125, -1.925854206085205, -0.5961701273918152, -0.30283400416374207, 0.4501618444919586, 0.8080061674118042, 0.30985456705093384, 0.7499703168869019, 2.2913358211517334, 0.8286400437355042, -1.0692217350006104, -0.3046892583370209, -0.857214629650116, -0.422626793384552, 0.7823188900947571, 0.5207217335700989, 0.423393189907074, 1.259224772453308, -0.46931377053260803, -0.12839151918888092, 0.18484635651111603, 0.21762484312057495, 0.44399335980415344, 1.3804749250411987, 0.6951119303703308, -0.5859859585762024, 0.31875506043434143, -0.0644037127494812, 0.4530889689922333, 1.224212646484375, 0.41408035159111023, 1.4608900547027588, -0.0857604369521141, -0.9569324254989624, -0.587196409702301, 1.3001412153244019, -0.31285253167152405, -0.3042261004447937, -1.1998887062072754, -0.48621320724487305, -0.027072187513113022, -0.31971845030784607, -0.6836655735969543, 0.6033797860145569, -0.32282280921936035, 0.7850947976112366, 0.5905160903930664, 1.2518483400344849, -0.28390583395957947, 1.856444239616394, -1.2907975912094116, 0.49789708852767944, 1.1303956508636475, 0.8240662217140198, -1.2002313137054443], [0.3351537883281708, -0.4045884609222412, 0.5192806720733643, -1.6371277570724487, 1.7533533573150635, 0.47165337204933167, -0.08411264419555664, 0.3175172805786133, 0.5866120457649231, 0.5396068692207336, -0.6747642755508423, 1.7891316413879395, -0.9983135461807251, 0.2937466502189636, -0.982384979724884, -1.8964860439300537, -0.3595772981643677, -0.19630660116672516, -0.032532837241888046, -0.9116479754447937, 0.5998305082321167, 1.802575707435608, 1.4279154539108276, 0.17426683008670807, -0.04268409311771393, 0.8455460667610168, 0.6330937147140503, -0.7982037663459778, -1.195054292678833, 0.31572282314300537, 0.1651185005903244, -0.7640838623046875, 0.1574908196926117, 0.971391499042511, -0.942224383354187, -2.9186899662017822, -0.1106443703174591, 0.1833615005016327, -1.4263907670974731, 1.4718716144561768, -0.03683997690677643, 0.39364010095596313, 1.4211516380310059, -3.0811853408813477, -1.4162427186965942, 0.9059451222419739, -0.09099528193473816, 0.5513193011283875, -0.7873890399932861, 0.5665884017944336, -0.710048258304596, -0.4125019609928131, 2.2488253116607666, -0.35982850193977356, -0.41156700253486633, -0.48670175671577454, 1.574549674987793, -0.13727234303951263, 0.7714179754257202, 0.8101927042007446, -0.18888400495052338, 0.4319865107536316, 0.6803012490272522, -0.3955068588256836, -0.6768417954444885, -0.008809675462543964, -0.747843325138092, 1.1194325685501099, 0.12993508577346802, -1.9819793701171875, 1.7237412929534912, 0.8165831565856934, 0.8171820640563965, 0.9394868612289429, 0.3390286862850189, -0.9590614438056946, 1.5792791843414307, -1.8532685041427612, -1.9325374364852905, -1.8498249053955078, 0.7109400629997253, -0.31761279702186584, -0.652815580368042, 0.06871302425861359, 1.1982471942901611, 0.2689801752567291, 0.015038404613733292, 0.7883567214012146, 0.38181450963020325, 0.8300371170043945, -1.022540807723999, 0.3263801634311676, 0.4562442898750305, 1.3384244441986084, -0.5458101034164429, 0.9078553318977356, -1.5922669172286987, 0.30085399746894836, -1.2913269996643066, -1.4493485689163208, 0.18312783539295197, -0.4908112585544586, 0.5609988570213318, 1.214134931564331, -0.6493780016899109, -1.359645128250122, 0.6412121653556824, 1.1944174766540527, 1.1175116300582886, -0.2994242310523987, 0.4929669201374054, -1.56086266040802, -2.475221633911133, -0.32142457365989685, 1.5306135416030884, 1.2339353561401367, 0.1422392725944519, 0.1254478096961975, 0.11758557707071304, -1.2361520528793335, 1.3021609783172607, 0.03791830316185951, 0.6102601885795593, -1.5183969736099243, -2.0040078163146973, -0.13970144093036652, -0.16084952652454376, -0.34822776913642883, -0.05674770846962929], [-1.1945348978042603, -0.5753253698348999, 0.25804007053375244, 0.5633162260055542, -1.1117063760757446, 1.4040106534957886, -0.12927234172821045, -0.23133111000061035, 1.670414924621582, -0.6746127605438232, 1.4483273029327393, 0.5267781019210815, -0.4926586449146271, 1.0171267986297607, 0.9588921070098877, 0.17812089622020721, -0.0579192191362381, -1.009355068206787, -0.5539770126342773, -1.7700414657592773, -1.337545394897461, -2.538867712020874, -0.49776092171669006, 0.848076343536377, -0.23051802814006805, -0.12641215324401855, 1.6466413736343384, -0.4934854209423065, -1.6876851320266724, -0.18241654336452484, -0.790032148361206, 1.4574116468429565, 0.3737742006778717, 0.45481473207473755, 1.7721549272537231, -0.6666590571403503, -0.6761626601219177, 1.0349286794662476, -0.6071538329124451, 0.31628650426864624, 0.5508231520652771, 1.8052083253860474, 1.1450464725494385, -2.720844030380249, 2.206613063812256, -0.5474668741226196, 0.6814975738525391, 0.31630927324295044, -0.9844047427177429, -0.5107887387275696, 0.2790970206260681, -0.7344367504119873, 0.5891379117965698, 0.9328216910362244, 0.37850943207740784, -1.095902442932129, 1.0578054189682007, 0.6395545601844788, -0.005221860948950052, 0.545088529586792, -0.7938389778137207, -2.032987594604492, -0.78714519739151, 0.5125420093536377, 1.0107544660568237, -1.4260437488555908, -0.19392624497413635, 0.47303837537765503, -1.3008354902267456, 0.39287668466567993, 0.24897760152816772, -0.9201187491416931, -0.44962286949157715, -0.46354854106903076, -0.8948367834091187, 2.320786237716675, 0.13905413448810577, -0.19188059866428375, -0.7727500200271606, -0.46143895387649536, 0.717893660068512, -0.015318915247917175, 1.0227711200714111, -0.39797842502593994, -0.02689463645219803, -0.16431757807731628, 0.4905965030193329, 0.2633056342601776, -1.8383287191390991, 1.1493029594421387, 2.148113489151001, 0.9802846908569336, 0.5971589088439941, 0.49469852447509766, 0.8946643471717834, -1.3347939252853394, 0.2041480988264084, -0.04212302342057228, 0.1520538181066513, 0.0025459728203713894, -0.8877604603767395, 0.000510254583787173, 2.547525644302368, 0.7930834889411926, 1.7224565744400024, -0.7612877488136292, 0.01436510682106018, 0.7002500891685486, 2.509554862976074, 1.7483841180801392, 0.8223876357078552, -0.3471299707889557, 0.04011067375540733, 0.27569645643234253, 0.6254921555519104, -0.17101126909255981, -0.8739950656890869, 0.46754348278045654, -0.47216668725013733, -1.6629208326339722, 0.8390924334526062, 0.7273308634757996, -0.16394737362861633, 0.5841933488845825, -0.4439648389816284, -1.6903825998306274, -1.6740093231201172, -0.8529303073883057, -0.5272142291069031], [0.2754257321357727, -0.68653804063797, -0.5331925749778748, 0.4526456594467163, 1.3192311525344849, -0.3198055028915405, -0.9647615551948547, 1.1085525751113892, -0.30414602160453796, -0.6873751878738403, -0.5394529104232788, 0.4251241087913513, 1.132908821105957, -0.5374106168746948, -1.031130313873291, 0.029225945472717285, -1.0260027647018433, -0.07065027207136154, 0.745551586151123, -1.0866942405700684, -1.159472107887268, -1.2648751735687256, 0.42145583033561707, 0.4288378357887268, 1.0750505924224854, -0.6760867834091187, -1.176814317703247, -0.42374253273010254, -0.8644254803657532, -1.60343337059021, 1.0934453010559082, -1.4785534143447876, -0.24184291064739227, 0.4870782494544983, 1.2464840412139893, -0.03767488896846771, -0.5718315839767456, 0.42818501591682434, 0.0068588051944971085, 1.5906286239624023, -0.4130450189113617, -0.007933154702186584, -1.2171053886413574, -1.2432861328125, -0.21335549652576447, -0.08215431123971939, -0.8174020648002625, 0.5861817002296448, -0.681440532207489, 1.286760687828064, 1.0223584175109863, 0.4212356507778168, -0.2899639904499054, -0.25259557366371155, -0.23237819969654083, 2.226685047149658, -0.6534450054168701, -0.40501466393470764, 0.5043330788612366, 0.8721621036529541, 0.30442583560943604, -0.08606840670108795, -0.7163158655166626, -0.9150736927986145, -0.9418379664421082, 0.8338346481323242, 0.9527292847633362, -0.6227689981460571, -0.38633203506469727, -0.3589041531085968, -1.1659423112869263, -0.011877404525876045, -0.14029426872730255, 0.006602453999221325, 0.2747199833393097, 1.882568597793579, -0.7203407883644104, 0.20911258459091187, 0.9165267944335938, -0.9175760746002197, 0.12145054340362549, 0.3760555386543274, -1.0111500024795532, -1.7485719919204712, 0.44207295775413513, 0.29153165221214294, -0.08607623726129532, 1.8761358261108398, -0.4187721312046051, -0.48889076709747314, -2.375901460647583, -0.4122714102268219, 0.2896706461906433, 0.4783061444759369, -1.4898203611373901, 0.8593286275863647, 0.10180624574422836, -0.4940028190612793, -1.2336863279342651, -0.6062356233596802, 0.6453698873519897, -0.110559843480587, -0.03521765395998955, -1.6009455919265747, -0.14398808777332306, 0.2407853752374649, 2.113252878189087, 1.2914493083953857, -0.009522230364382267, -1.3474929332733154, -1.1369974613189697, 1.3110847473144531, -0.5225940942764282, 0.5935909152030945, 0.3924841284751892, 2.215226650238037, 0.4628559350967407, 0.7920442819595337, -1.1611324548721313, -0.32550081610679626, -0.9820075631141663, 0.4970397651195526, 0.5336697101593018, -0.7575297951698303, 0.17616812884807587, -1.2644691467285156, -1.9200071096420288, 0.553011953830719, 1.5194865465164185], [-1.3073899745941162, 0.2543855309486389, -1.6690449714660645, -0.9611058235168457, 1.4118562936782837, 0.07907130569219589, -0.24214430153369904, 0.19475753605365753, -1.6547527313232422, -1.1177692413330078, -0.6168622374534607, 0.5355113744735718, -0.1893644481897354, 0.14586247503757477, 0.3182213008403778, 0.9702228903770447, -0.7897189855575562, -0.10232261568307877, -0.5133396983146667, -2.8051908016204834, -1.1803250312805176, 1.031226396560669, -0.11129750311374664, 0.9570363759994507, 0.4186394214630127, -0.7748543620109558, -0.41022536158561707, -1.058227300643921, -2.0517101287841797, 0.31706708669662476, -2.1403214931488037, 0.08671700209379196, -0.6810049414634705, 0.627052903175354, -0.2791326642036438, -0.17786116898059845, -0.35301607847213745, -1.071232557296753, -0.8535609841346741, 0.4744246006011963, -0.9062385559082031, -1.0091733932495117, -1.4858239889144897, 2.3790547847747803, 0.17947009205818176, 1.8761272430419922, 0.5379440784454346, -0.2819748818874359, -0.7925296425819397, -0.513363242149353, -0.711482048034668, 0.11201241612434387, 0.680020809173584, 0.23564766347408295, -1.7569535970687866, -0.8781991004943848, 0.937656581401825, 0.08079110085964203, 0.056863315403461456, 0.010025362484157085, -0.0801781564950943, 1.4617037773132324, -0.293412983417511, 0.08535660803318024, -1.6928986310958862, -0.23580749332904816, 0.4963681995868683, -1.0101665258407593, 0.30616453289985657, -0.5614936947822571, 1.2603955268859863, 0.6303557753562927, -1.4164847135543823, -0.04052191600203514, -1.410571575164795, -0.8046948909759521, 0.6650586128234863, 0.32492518424987793, -2.8960049152374268, 0.5528899431228638, 1.5869147777557373, 1.091711401939392, 0.10482460260391235, 0.279342919588089, 0.23708423972129822, -1.3839291334152222, 1.6990281343460083, 1.5133644342422485, 1.2106231451034546, -0.7844175696372986, -0.2546736001968384, -0.09682413935661316, -0.2686432898044586, 0.2694058418273926, -0.3922923505306244, 0.2811126410961151, -0.18253597617149353, 0.5289528965950012, 0.5374950766563416, 1.4766584634780884, 0.16348570585250854, -0.108472540974617, -2.4016518592834473, 1.7592798471450806, 0.8370874524116516, -0.03278452903032303, -0.5655945539474487, -1.8887962102890015, -1.293461799621582, 0.8189147114753723, -1.8553247451782227, 2.231663942337036, 0.309068500995636, 0.14474515616893768, 0.08067847043275833, 1.157896637916565, -0.9029330611228943, 0.027227304875850677, 0.6288476586341858, -0.90120929479599, -0.7677125930786133, -2.653569459915161, 0.10625283420085907, -1.554269552230835, 0.6844798922538757, -1.2321455478668213, 0.06386273354291916, -0.028838524594902992, 0.6179941892623901], [-0.473301500082016, -0.9129246473312378, -1.0436819791793823, -1.6029891967773438, 1.431947112083435, 1.2098723649978638, 0.8220398426055908, 0.3376237750053406, 0.6107345819473267, -0.5276131629943848, 1.2820351123809814, 0.3319573402404785, 0.12403128296136856, -0.5485497713088989, -1.2851386070251465, 0.19100484251976013, 1.2061837911605835, -1.340266227722168, -0.9572451710700989, -0.19803239405155182, -0.21573443710803986, -1.5095330476760864, -0.3665616810321808, -0.008046768605709076, 0.33146777749061584, 0.776850163936615, -0.6145762801170349, 1.3713366985321045, 1.4228630065917969, 1.1477898359298706, 0.09764227271080017, -2.134913921356201, -0.4056570827960968, -0.10999910533428192, -0.6572523713111877, -0.1824462115764618, 1.5475507974624634, 1.9983839988708496, -0.1598455309867859, -0.4051062762737274, -1.9026312828063965, 0.9141255617141724, -0.9675923585891724, 0.3230060040950775, -0.44177013635635376, 3.231491804122925, 1.5752027034759521, -1.2469838857650757, 0.5298680067062378, -0.39813560247421265, 0.16607339680194855, 0.9091253280639648, -0.20814985036849976, -1.9841028451919556, -0.5750985145568848, -1.1455888748168945, -0.8263487219810486, 0.6381859183311462, 1.1302798986434937, -1.0991573333740234, 0.5712634325027466, 0.08937259763479233, -0.7157418727874756, 0.24265047907829285, 0.33437204360961914, -0.36361590027809143, -0.8719109892845154, -0.12962943315505981, 0.24206037819385529, -0.9461037516593933, 1.2208424806594849, -1.148402214050293, -0.2739122211933136, -0.08632438629865646, -0.09120335429906845, -0.2544059455394745, -1.5420022010803223, 0.42594295740127563, -1.0000805854797363, 0.8928447961807251, 0.5786361694335938, -2.2113401889801025, -0.10678578168153763, -1.9542834758758545, 0.5647832155227661, 0.8980168104171753, 1.5224522352218628, 1.708333969116211, 0.1485661268234253, -0.3249160349369049, -0.967552661895752, -1.6549032926559448, 1.4753754138946533, 0.6428022384643555, -0.3222818672657013, -0.3691136837005615, -2.6338143348693848, 0.8261895179748535, 0.2908725440502167, 1.3951311111450195, -0.10853273421525955, 0.5613729953765869, 0.3285634219646454, -0.04244859144091606, -0.03168991208076477, -0.3286885917186737, 0.26347118616104126, 0.7468286752700806, -0.32607585191726685, -1.3107088804244995, -0.9728098511695862, -0.9102500081062317, -0.6744248270988464, -0.4797731637954712, 0.9740585088729858, 1.5057278871536255, 0.2776945233345032, 1.4683891534805298, -0.7217587232589722, -0.8612306118011475, -2.0501229763031006, -0.16149505972862244, -0.4494612514972687, -1.658383846282959, -0.5581780076026917, 0.8066848516464233, -0.5205872654914856, 0.20121608674526215, -1.5885069370269775], [0.624400794506073, -1.4054688215255737, -2.826549768447876, 0.311482310295105, -0.9687297344207764, 1.60299551486969, 0.34048449993133545, 0.06913407891988754, -0.023336485028266907, 0.1216493472456932, 0.03266060724854469, -1.2014614343643188, 0.8760334849357605, 0.4700693190097809, -0.5988824367523193, 0.4960753917694092, 1.865755558013916, -1.3757531642913818, -0.05104469135403633, -0.7906039953231812, 1.3821858167648315, 0.8538599610328674, -0.40172454714775085, -1.5994303226470947, 0.1215951144695282, 1.366407036781311, -1.3930913209915161, -0.3577515184879303, 1.78567636013031, 1.9716355800628662, -1.1537986993789673, 0.34618231654167175, 2.1288390159606934, 1.1986753940582275, -1.052168369293213, -2.145284652709961, -1.5097408294677734, 0.282668799161911, -0.15904994308948517, -0.822887659072876, 0.666628360748291, -0.05598791688680649, 1.7426469326019287, 0.8445577025413513, 1.145916223526001, -0.9331396818161011, 0.18928182125091553, -1.8357105255126953, 0.6425512433052063, 0.12085897475481033, 0.3282833993434906, -1.15812087059021, -1.583553671836853, 0.46433761715888977, 0.8614020347595215, 0.7183054089546204, -1.4109362363815308, -0.8998084664344788, 0.008479825221002102, 0.5336576104164124, 1.6928905248641968, 0.8725316524505615, 0.21847203373908997, -1.204910397529602, -0.5745800733566284, 0.5686526894569397, -0.9503616690635681, -2.144390344619751, -0.1767946034669876, -0.013933717273175716, -0.6439235210418701, -0.2579856812953949, -0.0877579003572464, 0.10296273231506348, 0.5216081738471985, 0.31083324551582336, 0.6946093440055847, 1.696946144104004, 1.516674518585205, 0.18946228921413422, 1.287671446800232, 0.6048675179481506, -0.38775768876075745, 1.0498062372207642, 1.0177429914474487, -0.2144281566143036, 0.7337768077850342, -0.06728792935609818, 0.26416119933128357, -0.5229097008705139, 0.6621010303497314, 0.1194523498415947, 0.5866444110870361, 0.7304169535636902, -0.5105726718902588, -0.11830955743789673, 1.5670599937438965, 0.05665699392557144, -1.4542268514633179, 1.3868048191070557, 0.5104423761367798, -1.4920862913131714, -1.268269419670105, -1.2920622825622559, -0.8146421909332275, 0.1439996212720871, -1.0556657314300537, -0.05565647408366203, 0.2218930721282959, -0.10362059623003006, -1.3824304342269897, 1.8039746284484863, -0.042226046323776245, -0.21611234545707703, -1.0594481229782104, 0.0726352259516716, -1.3022392988204956, 0.6261727809906006, -0.7941240072250366, -2.079728603363037, -0.40551117062568665, 0.33745625615119934, 0.4983688294887543, -1.1415975093841553, 0.40688595175743103, -1.3914355039596558, -0.5688035488128662, -2.1579971313476562, -0.22669948637485504], [-0.6367250680923462, 0.1994061917066574, -0.5093562602996826, 2.1925876140594482, 1.181280255317688, -0.11688996851444244, 0.5789534449577332, 0.455472856760025, 0.9957579970359802, -0.5995339751243591, -0.3974512815475464, -1.7503561973571777, -0.019085928797721863, 1.032778263092041, 0.8592889308929443, 1.4746509790420532, -1.0428187847137451, 1.6527739763259888, 1.2510228157043457, 1.6342681646347046, 0.46094822883605957, -0.27438655495643616, -0.34160399436950684, 0.3934534192085266, -0.03778194263577461, -0.11152081936597824, 0.9024009108543396, 0.23299898207187653, -0.9857832193374634, 0.27348142862319946, -0.9562017917633057, 1.4004745483398438, 0.262851357460022, 0.1981724202632904, -0.8435256481170654, 0.5343706607818604, -1.732362985610962, 0.011042811907827854, 1.7619563341140747, -1.4984571933746338, 0.12462662160396576, 0.9004639387130737, -0.48310017585754395, 0.6857883930206299, -0.8186402916908264, 0.11053958535194397, 0.8800076842308044, 0.21644853055477142, -0.8294157385826111, 0.6570689082145691, -1.0228594541549683, -0.5773659348487854, -0.04871732369065285, 1.3389121294021606, 0.4969557523727417, 1.4889739751815796, -0.762309193611145, 1.1317960023880005, -0.2706991136074066, -0.17418822646141052, 0.11757272481918335, 2.482614517211914, 0.054138537496328354, -0.38156139850616455, 0.8979007005691528, -0.40022096037864685, -0.8870865106582642, -0.7381455898284912, 0.4803329110145569, -0.6107782125473022, -2.7360708713531494, 1.8262077569961548, 0.32696154713630676, -0.9809556603431702, -0.049219705164432526, -0.681053876876831, 1.2325679063796997, -0.8155319094657898, 0.7819867730140686, 0.7938743829727173, -1.6348317861557007, 0.4031815826892853, -0.7942801713943481, -1.2064743041992188, 1.3055682182312012, 0.03817310929298401, -0.26889288425445557, -0.039132993668317795, 1.124693751335144, -0.49808982014656067, 0.17176491022109985, -0.7111977338790894, 0.5927702784538269, -0.799313485622406, 0.28697776794433594, -1.3748589754104614, -0.4945954978466034, -1.0050749778747559, -0.7826741933822632, -0.8196744918823242, -1.6104841232299805, 1.6908683776855469, 0.4036266803741455, -0.516868531703949, 1.1326119899749756, 0.15902699530124664, -1.118092656135559, 0.16223755478858948, -0.777327835559845, 0.5354480147361755, 0.3905054032802582, -0.8725588917732239, 0.6405038833618164, 0.3718123733997345, 0.792633056640625, -0.9869529008865356, -2.040544033050537, -0.7095918655395508, 0.07442492991685867, -1.874466061592102, 2.040468692779541, 0.2842992842197418, -0.3316113352775574, -0.8812013864517212, -0.835910975933075, -0.2990926504135132, -0.37355470657348633, -0.552848219871521, 1.6159082651138306], [0.9520860910415649, 2.011192798614502, -2.2074851989746094, 1.7964880466461182, -0.3086279332637787, 1.5496721267700195, -2.1824164390563965, 0.5700634717941284, -0.43415337800979614, 2.3287105560302734, 1.5623873472213745, -0.542662501335144, 2.3395042419433594, 2.139249086380005, -1.6255202293395996, 1.1082926988601685, 0.8081346154212952, 0.8417335748672485, 0.29727864265441895, 0.30940237641334534, 1.036074161529541, 0.5519532561302185, 1.2625890970230103, 3.2846317291259766, 1.8502418994903564, 0.9407621622085571, 0.4732712209224701, -0.12143032997846603, -0.6401566863059998, -1.9079731702804565, 0.48685717582702637, 0.6382666826248169, 1.6173003911972046, 0.9161478281021118, 1.1595784425735474, -0.8443003296852112, 1.0917552709579468, -1.0440183877944946, -0.06473979353904724, 1.7416720390319824, 0.36941224336624146, 0.6465833187103271, -1.7551283836364746, 0.6058976650238037, -0.6679855585098267, 1.050208330154419, 1.4097539186477661, -0.5435770750045776, -0.23473180830478668, 1.209721565246582, 1.7071254253387451, 1.0198410749435425, 2.0552215576171875, -0.039229828864336014, -0.442209392786026, 0.7086904644966125, -1.0516608953475952, -0.22159160673618317, -0.030417509377002716, -0.2371591180562973, 1.1909326314926147, 1.4092222452163696, 0.8157207369804382, -0.6448469161987305, -0.14648854732513428, 0.7083922028541565, 0.8133350014686584, -0.4821629524230957, 0.5521742701530457, 1.1196213960647583, -0.013376828283071518, -0.6780292987823486, 0.6457138657569885, 1.5493544340133667, -0.49719947576522827, -0.41496986150741577, -1.6914517879486084, 1.0607060194015503, -1.4908760786056519, 0.229519784450531, 0.145277738571167, -1.5294908285140991, 0.8965591788291931, 1.1903786659240723, -0.11285872757434845, 2.4051952362060547, 0.9931066036224365, 0.7036466598510742, -0.5244544744491577, 1.4423377513885498, -1.7760530710220337, -0.9934412240982056, -0.23306965827941895, -2.5465707778930664, -0.7612818479537964, 2.364346742630005, 0.2563032805919647, 0.1464601755142212, -0.20278717577457428, 0.3978461027145386, 0.21122398972511292, 0.6934573650360107, -1.6946091651916504, -0.43733593821525574, -0.17571982741355896, -0.5316144824028015, 0.8852013945579529, -0.1435754895210266, -2.3740501403808594, -0.38589534163475037, 0.7826381921768188, 1.6138345003128052, -0.12752538919448853, -1.0083928108215332, 0.12406975775957108, 0.08583677560091019, 1.024404525756836, 1.251150369644165, -0.3254733383655548, 0.6945326924324036, -2.2240350246429443, -0.9460142254829407, -1.6273142099380493, 0.22517448663711548, -0.15800802409648895, 0.004063733387738466, -0.1409400999546051, 0.592832624912262, 0.5219555497169495], [0.5873674750328064, 1.1389943361282349, 0.2552388608455658, 0.39630359411239624, -0.39164525270462036, 1.246120810508728, 1.5995887517929077, 0.10108458250761032, 1.1754467487335205, -2.757857322692871, 0.16304665803909302, 0.6206491589546204, -0.29445958137512207, 0.36637455224990845, -0.6870123744010925, 0.1337496042251587, 0.5720307230949402, -1.4365990161895752, 0.08736461400985718, -0.27101197838783264, 2.569021463394165, -0.18417127430438995, 1.4379891157150269, -0.13617081940174103, -1.6862921714782715, -1.1903382539749146, -1.9323452711105347, -0.92673659324646, -1.5811713933944702, 0.13076162338256836, -0.8168264031410217, -0.11819367855787277, 0.5388873219490051, 0.6304168105125427, -0.6190957427024841, -0.3441223204135895, 0.2132336050271988, -0.16352936625480652, 0.3697054386138916, -2.0752475261688232, 1.5959464311599731, 1.974158763885498, -0.6569926738739014, 1.3388986587524414, 0.40471354126930237, 0.9653527140617371, -0.2650146484375, 0.3462376594543457, 0.9117892980575562, 1.6284986734390259, -0.00940724741667509, 0.6213732957839966, -0.4578295648097992, -1.0230110883712769, 0.5053877830505371, 0.6045452356338501, 0.18316371738910675, -1.577933669090271, -0.9275983572006226, -0.9219717979431152, -0.659435510635376, 0.2497810274362564, 0.33145418763160706, -0.07081680744886398, 0.1089460700750351, -0.06639302521944046, -0.8908079862594604, 1.2746142148971558, 0.6077418923377991, 0.5006638765335083, -0.4675960838794708, -0.18865051865577698, -0.5025953054428101, -1.4623165130615234, -1.8213871717453003, 0.3090873658657074, -0.3734070062637329, 0.3414144814014435, 1.135184407234192, -0.9311378598213196, 0.8610410690307617, 0.12708628177642822, 1.2815650701522827, -0.7990378737449646, -0.2244618982076645, 1.0746487379074097, 0.4586767554283142, 0.920566201210022, 0.30825042724609375, -0.2017323225736618, -0.6724968552589417, -1.2127808332443237, 0.7627785205841064, 1.2847052812576294, 0.6471256613731384, 1.0505067110061646, 0.09802506864070892, 0.5528842210769653, 0.5070292949676514, -0.5205000638961792, 2.1753740310668945, 0.9270008206367493, 1.0458709001541138, 0.40256744623184204, -0.47271299362182617, -0.2145056277513504, -0.04977500066161156, -0.11959601193666458, 0.3412508964538574, -1.6432623863220215, -2.2976293563842773, 0.19364196062088013, 0.04350727051496506, 1.6661806106567383, -0.9025113582611084, -0.039191748946905136, -0.13251036405563354, -1.1694982051849365, 1.214332938194275, 0.8366081714630127, -0.39110395312309265, -0.3278907537460327, 1.5287574529647827, -0.38020750880241394, -0.16592958569526672, -0.13065530359745026, -0.15018154680728912, 0.32983750104904175, 1.3508943319320679], [0.9107475280761719, -0.8281428217887878, 2.044752359390259, 0.11323660612106323, 1.6521317958831787, 1.815794587135315, -1.272337555885315, 1.022462248802185, -0.6178603172302246, -1.3120217323303223, -0.8658444881439209, -1.1487270593643188, -0.8948954939842224, 0.2475123107433319, -1.967160940170288, 0.4449988901615143, -0.7059466242790222, 0.6435129642486572, 0.6032631397247314, -0.19241264462471008, -0.547619104385376, -1.9501179456710815, 0.14399100840091705, -0.6621123552322388, 0.16210739314556122, 0.02739652618765831, -1.3983309268951416, -0.7876075506210327, -1.4948087930679321, 1.411120891571045, 0.2486751526594162, -0.8837896585464478, -2.541746139526367, -0.7811751365661621, -0.7995145916938782, -1.5946519374847412, -0.48863959312438965, 0.3389756977558136, -0.4862850308418274, -0.4114653170108795, -0.14658452570438385, 0.27242758870124817, 1.7771201133728027, -1.0215435028076172, 0.7794063091278076, -0.3718024492263794, 0.5017963647842407, -1.2211395502090454, -0.3347875773906708, -1.0584001541137695, -0.5258565545082092, 0.4371113181114197, 1.236446499824524, -1.3970451354980469, -0.10484177619218826, 0.27142634987831116, -0.0887824296951294, 1.1929028034210205, -1.2690036296844482, 0.9267600178718567, 0.7500330805778503, -0.37282833456993103, -0.6615960001945496, -0.7444427609443665, 0.4093043804168701, 0.5323149561882019, -0.8405918478965759, -0.42907482385635376, -0.4596250355243683, 0.5343091487884521, 1.3609610795974731, 0.2920490503311157, 0.6745732426643372, -0.634519636631012, 0.16938407719135284, 0.6276097297668457, -1.3983286619186401, -0.5193239450454712, -1.1722791194915771, 0.08135994523763657, 0.44034475088119507, -1.2911840677261353, 1.2643171548843384, 0.4024912118911743, 0.28168410062789917, 0.5016399025917053, -0.5046244263648987, 1.8154120445251465, -0.4594095051288605, 0.3625824451446533, 0.16949523985385895, -0.5437358021736145, -0.33330076932907104, -1.4031273126602173, 0.24337166547775269, -3.2743546962738037, 0.5049607157707214, 1.7655106782913208, 0.6861226558685303, 1.0778601169586182, -1.445933222770691, 0.17415496706962585, -0.9046975374221802, 0.27452221512794495, 1.488732099533081, 0.1525825411081314, -0.057124048471450806, 1.4166951179504395, -0.4175873100757599, 0.027342190966010094, -0.5048878192901611, -0.5699253678321838, 0.2651240825653076, 0.4130186438560486, -1.589293360710144, -0.6792101860046387, 0.04071807861328125, 0.16226717829704285, 1.044873833656311, -0.7365232706069946, -1.6817570924758911, 0.43109214305877686, 0.5654366612434387, 0.07999905943870544, 1.3138550519943237, 0.5729981660842896, 0.8227885365486145, 0.5117254257202148, 0.1507955640554428], [1.2001452445983887, -0.655937671661377, 0.14263299107551575, -0.485988974571228, -0.950476348400116, -1.4210764169692993, -0.20505966246128082, -0.8229893445968628, -0.9765655398368835, -0.8810511231422424, 1.0213021039962769, -0.8608747720718384, -0.8338874578475952, 0.020623957738280296, 0.061085984110832214, 0.9185018539428711, -2.4344077110290527, 0.26909416913986206, -1.144066333770752, 0.8566271662712097, 1.763216495513916, -0.8459583520889282, -0.2663235664367676, -1.3086045980453491, 2.325016498565674, 2.3640377521514893, -0.14713765680789948, 0.8915144205093384, 0.49848413467407227, 0.1809595376253128, -1.7709764242172241, -1.2428090572357178, -0.9794386625289917, 0.3911013603210449, 0.4530567228794098, 0.1275264173746109, 0.011477584019303322, 1.0950696468353271, -0.05321585386991501, 0.7319950461387634, 0.9174445271492004, -0.12218674272298813, 0.9395143985748291, 1.3733559846878052, -1.5103121995925903, 0.6056406497955322, -1.086023211479187, 1.5182218551635742, 0.24350270628929138, 0.5116889476776123, 0.032146621495485306, -0.4885022044181824, 1.1312147378921509, -1.4391539096832275, 0.8553005456924438, 0.8928017020225525, -1.583767294883728, -0.49825868010520935, -0.1974821239709854, -0.7582363486289978, -0.4205319583415985, 1.016494870185852, 0.4710993766784668, 1.524355411529541, 0.2924334406852722, 1.1202099323272705, 1.6057329177856445, -0.551307737827301, -0.49375471472740173, 0.1556461602449417, -0.863340437412262, 0.33511266112327576, 0.8415584564208984, -0.7545217275619507, -1.9271516799926758, -0.10046656429767609, -0.14444464445114136, -0.23561209440231323, 2.0241899490356445, -1.8623274564743042, 0.11347977817058563, 1.7988210916519165, 0.2400209754705429, -0.7865990996360779, 2.145693063735962, -1.5299755334854126, -0.6836608052253723, 1.7712074518203735, -1.1311696767807007, -0.6023851633071899, -0.00033294878085143864, 1.550371766090393, -0.3703869581222534, -1.1629103422164917, 1.1001249551773071, -0.7031491994857788, 0.9548141360282898, 1.6770416498184204, 1.8048906326293945, 0.278485506772995, 0.22990809381008148, 1.4180424213409424, 0.39411410689353943, 0.048050373792648315, 0.8364645838737488, 0.25215473771095276, 1.4365354776382446, 1.036444902420044, 0.5874963998794556, 0.49625667929649353, -0.17208945751190186, -0.5711377859115601, 0.839236855506897, -0.03769372031092644, -1.813568353652954, 0.43787825107574463, 0.31052646040916443, 0.26081332564353943, -0.6114970445632935, -2.0918869972229004, 0.44302812218666077, 0.5534666180610657, 0.29068297147750854, -1.7267082929611206, -0.5846365094184875, -0.02508670836687088, -0.10873127728700638, 2.4109203815460205, -1.629433274269104], [0.6783615350723267, -0.33108773827552795, 1.1014865636825562, -0.9976310729980469, 1.0188583135604858, 0.9768897891044617, 1.5574864149093628, -1.5639511346817017, 2.2553460597991943, -0.09665640443563461, -0.20300565659999847, 0.5379378795623779, 0.06502191722393036, -0.7876924872398376, -0.4978601932525635, 1.6275736093521118, 1.2751654386520386, 0.07348798960447311, -2.9699954986572266, 1.7127481698989868, 0.36088812351226807, 0.9034132361412048, 0.04497947916388512, 0.20546099543571472, -0.19192887842655182, -1.8896183967590332, -0.17138762772083282, -0.5604336857795715, 0.39609822630882263, 0.4355544447898865, 0.7765408754348755, -0.5634142756462097, 1.578849196434021, -0.19882366061210632, -0.946485161781311, -0.7059746980667114, 1.9061064720153809, 0.35358762741088867, 0.6892683506011963, 1.2092523574829102, -1.2480392456054688, 0.3185957372188568, 0.7766856551170349, 1.342499852180481, -1.1188465356826782, 0.8568846583366394, -0.027018241584300995, -1.4271477460861206, -0.329174667596817, -0.7542897462844849, 0.9757561683654785, 0.354804664850235, -0.8355432748794556, -0.08554072678089142, 0.26507243514060974, -0.9248672723770142, 1.4102550745010376, 1.2606102228164673, -0.8090587258338928, 0.2180735170841217, 1.510449767112732, -0.5052103400230408, 0.02306763269007206, -0.03849206119775772, 0.04683121666312218, -0.8348736763000488, -1.4598215818405151, 0.8522290587425232, -0.4567180275917053, 0.678059995174408, 0.5441755056381226, 0.882835328578949, -1.5138494968414307, -0.6569206714630127, -1.225868821144104, 1.5458567142486572, 0.8526593446731567, 2.3928940296173096, 0.7832605242729187, -0.2520512640476227, 1.046305775642395, 0.7219354510307312, -0.37414008378982544, 1.5182714462280273, -1.374381422996521, 0.7791038751602173, -1.1396214962005615, -0.5002420544624329, 0.7336183786392212, -0.22867828607559204, -0.28416627645492554, -0.5159448385238647, 1.263789415359497, -0.0442071259021759, -1.9421746730804443, 1.7404710054397583, -0.2237035036087036, -1.1095383167266846, 0.751106321811676, 1.2790757417678833, 1.241959571838379, 0.7715767621994019, -0.5200747847557068, 2.317168951034546, 0.5902958512306213, -1.0007262229919434, 0.14252986013889313, -0.21127182245254517, 0.07492956519126892, -0.7187143564224243, 0.4956265389919281, 0.27510392665863037, 1.8645617961883545, 3.273421287536621, 0.9436231851577759, -1.3888466358184814, -0.3354860246181488, 0.9578720927238464, 0.9030722975730896, 1.1485882997512817, -1.5668718814849854, -1.5047390460968018, -0.42920926213264465, 1.9304572343826294, -1.0588148832321167, -0.8408757448196411, -0.5017116665840149, 0.03746277466416359, -0.11919830739498138], [-0.0727112889289856, 0.2912752032279968, -0.39353829622268677, -0.3999764323234558, 0.43932637572288513, -1.537482738494873, -0.4560288190841675, 1.5235817432403564, -0.4115830957889557, -0.632148027420044, 0.08651737868785858, -1.1642968654632568, 0.3856281340122223, 0.35961151123046875, 0.1492840200662613, -1.6183840036392212, 0.41754114627838135, -0.13717572391033173, -0.036663968116045, -0.6188786625862122, 1.2097235918045044, -1.5288292169570923, 0.3170763850212097, 0.5778155326843262, -2.519627094268799, 0.6958065032958984, 0.21488118171691895, -0.3564532995223999, -1.158023476600647, -0.338754802942276, 0.637140691280365, -0.1495550572872162, 1.7714194059371948, -0.16810248792171478, -0.3608000576496124, -1.0292915105819702, 0.8331261277198792, 2.676518201828003, 1.9556132555007935, -0.16885864734649658, 0.1473105549812317, -0.5356907248497009, -0.27946773171424866, 0.15921379625797272, 0.7845315337181091, -0.6503387689590454, -0.2747443914413452, 0.4255214035511017, 0.42131665349006653, -1.3038322925567627, 0.47395825386047363, -0.33842870593070984, -0.5385427474975586, -0.6245296597480774, -1.9175528287887573, -1.1070722341537476, -0.47595956921577454, -0.2964668273925781, 1.4326611757278442, 0.8409234881401062, -0.7929704189300537, 0.6820113658905029, 0.8896412253379822, -0.4565011262893677, 1.7334892749786377, 0.3489896059036255, -0.5611092448234558, 0.6359214782714844, -1.3636115789413452, 0.19632022082805634, 1.0427805185317993, 0.08938661217689514, 0.5994958281517029, -0.5017690658569336, 0.6003134250640869, -0.9268550872802734, 0.5353395342826843, -0.9002490043640137, -0.4289661645889282, 0.8325251936912537, -0.23387405276298523, -0.794903039932251, -0.7475128173828125, -0.06942950934171677, -0.7316150069236755, 1.0031318664550781, 1.1899302005767822, 0.7755372524261475, 0.5720286965370178, 0.8059622645378113, 0.7083913683891296, -1.8702185153961182, 1.0175118446350098, -0.36416754126548767, -1.8161892890930176, 0.8831700682640076, 0.15382641553878784, -0.007040667347609997, 0.1468949019908905, -0.28657469153404236, -0.5130436420440674, 1.1738749742507935, -0.976432740688324, -1.2962636947631836, 1.3868464231491089, -0.3132503628730774, 0.17385321855545044, 1.577292799949646, -0.9734461307525635, 0.29962417483329773, 0.34690362215042114, -2.188211441040039, -0.6545475125312805, -1.2280898094177246, 0.14801859855651855, -1.0396428108215332, 0.10708735883235931, 0.5497971177101135, -0.9343516230583191, 1.4141956567764282, -0.39790865778923035, 0.022807756438851357, -1.0701345205307007, 0.0729777067899704, -0.07436311990022659, -0.3043837547302246, 1.5365734100341797, -0.3378959000110626, -0.1562211662530899], [1.7332348823547363, -0.4581429064273834, 0.9644737243652344, -2.5201046466827393, -1.721432089805603, 0.5525343418121338, -1.9043680429458618, 0.9303316473960876, 0.06549893319606781, -1.1715819835662842, 0.4114041030406952, -0.37622830271720886, -0.17024901509284973, -0.42250868678092957, 1.271634817123413, -0.07334006577730179, -0.1360711008310318, 1.3121787309646606, 1.1406481266021729, 1.8837450742721558, 1.493573546409607, -0.994539201259613, -1.571797490119934, 1.9719687700271606, -0.42609703540802, 0.07303782552480698, 0.7580279111862183, -1.982365369796753, 0.832926869392395, 0.49518007040023804, -0.056793805211782455, -0.6273978352546692, -0.13336984813213348, 0.4747668206691742, -0.2830107808113098, -0.687466561794281, -3.668612003326416, 1.2738295793533325, 0.3304743766784668, -0.18661567568778992, -0.685981810092926, -1.3149561882019043, -0.8079610466957092, 1.1098229885101318, -0.5347819924354553, 0.2020338922739029, -0.2450510561466217, 0.8764716386795044, 0.05365768074989319, -0.6656246781349182, 2.1544477939605713, -1.0802887678146362, 0.0333508625626564, -0.3129137456417084, -1.7095471620559692, 0.5046848654747009, -1.0577974319458008, -0.7722324132919312, 0.9490271210670471, -0.2964268922805786, -0.9494150876998901, -0.6628380417823792, 0.5330219864845276, 1.7964993715286255, -1.973490595817566, 0.4740822911262512, 2.041046142578125, 0.1282745748758316, -1.7373191118240356, 0.416562557220459, 0.5001882910728455, -1.5531808137893677, -0.7095437049865723, -0.37899351119995117, -0.04328246787190437, 0.6711061000823975, 0.29460570216178894, -0.5722709894180298, -0.22526127099990845, 0.7552659511566162, -0.17879721522331238, 0.7197929620742798, 2.070579767227173, -0.060210440307855606, 0.4739817976951599, -1.228456974029541, -0.9595832228660583, 1.1571935415267944, 0.6530267596244812, 1.205093502998352, -0.3760707378387451, 0.5845960378646851, -2.4361913204193115, 0.1066916435956955, -0.49258530139923096, 0.2719901502132416, 0.7282402515411377, -0.1783219873905182, 0.3185400366783142, 0.2744564414024353, 0.4087245762348175, 0.6459656357765198, -0.16374917328357697, 0.20625615119934082, -1.6422983407974243, 0.6375442147254944, 0.9032048583030701, 1.1285881996154785, -1.1517512798309326, 0.7245848774909973, 0.6065578460693359, 0.5605056285858154, -0.5615835189819336, 0.8313413262367249, 1.470242977142334, 0.5349200367927551, 0.6587621569633484, 0.8887125253677368, 1.3145334720611572, 0.3262007236480713, -0.06789863109588623, -0.6157519817352295, -0.46486547589302063, 0.33509379625320435, -0.5822793245315552, 0.14858819544315338, -0.09995881468057632, -0.31377673149108887, 0.37013739347457886], [0.32706066966056824, -0.12176701426506042, 1.1290829181671143, 1.219915747642517, 0.10756031423807144, 0.014310439117252827, -1.3480799198150635, -0.9537366032600403, 1.7444480657577515, 1.3390003442764282, 0.5420578122138977, 1.2773165702819824, -1.4170708656311035, 1.417423129081726, -1.0942893028259277, 0.037805184721946716, -0.2210691124200821, -1.250491976737976, -0.01688683219254017, 0.478161484003067, -0.514033317565918, -1.8700995445251465, -1.3424729108810425, -0.7928900718688965, 0.9103775024414062, 2.477126359939575, -0.8936349153518677, -0.2236412912607193, 2.660644054412842, 1.5748130083084106, -0.7971099615097046, 0.5502991080284119, 0.6214650273323059, 0.34363657236099243, -1.3478076457977295, -1.0661134719848633, -1.2211449146270752, 0.03837297111749649, 0.9868808388710022, 0.18045051395893097, -1.3748725652694702, -0.48982128500938416, 0.15563321113586426, 0.8171272873878479, -0.617975652217865, -0.5942198634147644, 0.6977851986885071, -0.2762918174266815, 2.264084577560425, 0.5723650455474854, 0.053903959691524506, -1.7414555549621582, 0.690743625164032, 0.49127134680747986, 0.7282198071479797, -0.8516325354576111, 0.9110637903213501, 0.8561717867851257, 0.6246852278709412, 1.817602515220642, 0.08537139743566513, 0.6588497757911682, 0.5146326422691345, -0.2256654053926468, -0.34252363443374634, -0.04437199980020523, 0.19715528190135956, -0.9951003789901733, 1.762953758239746, -0.1817823350429535, 1.632137417793274, 0.29501718282699585, 0.11133241653442383, 0.9500743746757507, -1.0121904611587524, 0.6741935014724731, 0.11634863168001175, 1.3420398235321045, 0.17414841055870056, -0.7397052049636841, 0.04307871311903, 0.32586774230003357, -0.15045706927776337, -2.200084924697876, 1.8436297178268433, -0.5508484244346619, -0.5826494693756104, -0.1397940069437027, -0.5317043662071228, -0.692969560623169, -0.7645772695541382, -0.7368678450584412, -2.1446046829223633, 0.1163972020149231, -0.051983792334795, 1.2012028694152832, 1.5733370780944824, 0.6897644996643066, -0.7531659603118896, 0.3019627332687378, 0.7296274900436401, -0.46949300169944763, -1.1663421392440796, -1.0764806270599365, -1.1980562210083008, -1.190230131149292, -0.4185817837715149, 0.8539720177650452, -0.9612643718719482, 0.11674749106168747, -0.4877708852291107, 0.06689999252557755, 2.124734878540039, -0.6345866322517395, 0.4387858808040619, 0.23497740924358368, 1.0152078866958618, 0.8187830448150635, 0.2551244795322418, -0.12740755081176758, -1.4993597269058228, 0.931830644607544, -0.9042067527770996, -0.5936671495437622, -1.272925615310669, -0.868398129940033, -2.6326518058776855, 0.35397303104400635, 0.05292916297912598], [0.4572013020515442, 0.8396538496017456, -1.4697976112365723, 1.6239933967590332, -0.05614135414361954, -1.2592692375183105, -1.7304799556732178, -0.09676993638277054, -0.7791392207145691, 0.7745019793510437, -0.17282825708389282, 0.517130970954895, 1.4157164096832275, 1.7348945140838623, 0.16934022307395935, -0.7303984761238098, 0.8002254962921143, -0.41298380494117737, -0.08058620244264603, 0.0679154247045517, -1.2860913276672363, 0.7479493021965027, -0.1377224326133728, -0.9280235767364502, -0.344626247882843, 0.893771231174469, 0.7770504951477051, -0.378836989402771, -0.34214821457862854, -1.1422218084335327, -0.18930411338806152, 0.7592260837554932, -2.327237606048584, -0.7600806951522827, 1.1946245431900024, 0.6445013880729675, 0.4473114609718323, 0.46075013279914856, -1.8120590448379517, -0.728165328502655, -0.6041157245635986, 1.1058404445648193, 1.0868028402328491, -1.4706523418426514, 1.4766795635223389, -0.2064967006444931, 1.5093052387237549, -1.2787952423095703, 0.257887601852417, -0.8543274998664856, -0.4769047200679779, -0.6961799263954163, -0.7955683469772339, 0.20503956079483032, -0.2250576615333557, 0.537825882434845, -1.6131004095077515, -1.3406542539596558, -0.33159053325653076, 0.09359404444694519, 1.1183172464370728, 0.5876622200012207, -0.44351476430892944, -0.8761699199676514, 0.8267203569412231, -0.8270917534828186, 1.9635597467422485, -0.8651512265205383, -0.5015855431556702, -0.46957021951675415, -0.3948245644569397, 1.4065309762954712, 0.12994267046451569, 0.5593724250793457, 1.3938555717468262, -1.784911036491394, -0.5988163948059082, -0.02912965603172779, 0.6310837864875793, 0.5804638862609863, -0.9802056550979614, -0.32713213562965393, -0.740254819393158, -0.17629745602607727, 0.7596435546875, 0.8045297861099243, 1.8520866632461548, -0.9403550028800964, 0.9100772738456726, 1.816333293914795, -0.3413325846195221, 1.8613498210906982, -0.46296489238739014, -1.4064785242080688, 1.9389476776123047, 0.6013949513435364, -0.2517952620983124, -0.41867795586586, -2.1454355716705322, 0.3691372275352478, 1.2215979099273682, 0.5737338066101074, 0.817717432975769, -1.1974893808364868, 0.7164671421051025, 0.42760777473449707, 0.2425837367773056, 0.9722625613212585, 0.8236334323883057, 0.23332379758358002, 1.293200135231018, 0.4321209490299225, -0.2541957199573517, 1.4016551971435547, -0.5841359496116638, -1.1533479690551758, -0.5852688550949097, 0.06820423156023026, -0.33714911341667175, 0.9565756320953369, -0.9818928241729736, -1.2644819021224976, 0.6762187480926514, 1.467095136642456, 1.0084134340286255, 1.8503143787384033, 0.815200924873352, 0.6123435497283936, -0.7910652160644531], [0.3801995813846588, -1.8569098711013794, -0.4214870035648346, 1.0964984893798828, 0.04508449509739876, -0.7068053483963013, -0.9019604921340942, -1.160928726196289, 0.5916718244552612, -0.7743023633956909, -1.3283671140670776, -0.5306631922721863, 0.3658963739871979, -1.1238789558410645, 0.6049116849899292, 0.7850145101547241, 0.32667797803878784, -1.298943042755127, -0.86110520362854, -0.3962559103965759, 1.20512056350708, -0.46191146969795227, -1.186171531677246, 1.2497379779815674, 0.5636290311813354, 1.4932554960250854, -1.7243162393569946, -0.6844238042831421, 0.6172508597373962, 0.9998090267181396, 0.8767015933990479, 1.3538838624954224, 0.21075987815856934, -0.4648852050304413, -2.052659749984741, -0.655961275100708, -1.476684331893921, -0.7984521389007568, 0.6251179575920105, -0.7280909419059753, -0.19368012249469757, -0.3282756507396698, 0.10080818086862564, 0.4952089190483093, 0.9903682470321655, 0.46462804079055786, -1.9090670347213745, -0.7487478852272034, 1.979423999786377, 0.5490195751190186, -0.9427006244659424, 1.6342180967330933, -0.27870360016822815, -0.379882276058197, -0.278317391872406, -0.5883132219314575, -0.4149433672428131, 0.6309274435043335, 0.12294363975524902, 0.7820009589195251, -1.8941426277160645, 0.3279617726802826, 0.008318617008626461, 1.9461338520050049, -0.662484347820282, -1.3694748878479004, 0.1135864332318306, -1.0031944513320923, -1.5210658311843872, -1.1864268779754639, -0.27481240034103394, -1.2277617454528809, 0.7869426012039185, -0.9500212073326111, -0.2763386070728302, 0.8309158086776733, 0.5638159513473511, -0.6198097467422485, -0.3244921863079071, 0.4078127145767212, -0.1012839823961258, -2.3217289447784424, -0.31843671202659607, 0.9111585021018982, 0.3671947419643402, 0.5710145235061646, 1.9957716464996338, 0.8402179479598999, -0.9541304707527161, -1.2853376865386963, -0.617901086807251, -0.7315124869346619, -0.139754980802536, 0.17147096991539001, -0.22985218465328217, 0.12582410871982574, 0.5575743317604065, 0.6513399481773376, -1.3855575323104858, -0.4517726004123688, -1.3397786617279053, -0.067557692527771, 1.2510555982589722, -0.5534306764602661, 0.3611186146736145, 0.2925626337528229, -1.6314319372177124, 0.5183470249176025, -1.4557619094848633, 0.6265077590942383, 0.3778645694255829, 0.8837317228317261, -1.0542386770248413, 1.328515648841858, -0.5517803430557251, 0.187886580824852, 0.41382747888565063, -0.8220686912536621, 0.3441188335418701, 0.4366514980792999, -0.10519400238990784, 0.6007038354873657, -0.28005537390708923, 0.009879776276648045, 1.3446152210235596, 0.03448081389069557, 1.2886804342269897, 0.8231383562088013, -2.076108694076538], [1.6910144090652466, 0.6664595603942871, 0.713512122631073, -0.19186769425868988, 0.32907834649086, 0.1822880506515503, -1.5810229778289795, 0.04583460092544556, 1.6321367025375366, 0.30832743644714355, 1.3797038793563843, -0.6948133111000061, 0.3194628655910492, 0.218435138463974, -1.9983508586883545, -0.3618634343147278, 0.6778792142868042, 0.3951026201248169, -0.606665849685669, 0.371589720249176, -0.30290597677230835, 0.05298210307955742, 0.33406689763069153, -0.970565915107727, 1.6573634147644043, -0.4033760130405426, -0.17301836609840393, 1.5757473707199097, 0.20309245586395264, 2.022965431213379, -1.7109575271606445, -0.9849317073822021, -1.4792860746383667, -0.9260640740394592, -0.19729089736938477, -0.7543967962265015, -0.4366467595100403, -1.9911357164382935, 1.0155006647109985, 0.963524341583252, -1.2790106534957886, 0.14083705842494965, 0.6798386573791504, 1.4025789499282837, 0.07000281661748886, -1.602052927017212, 0.3929094076156616, 2.10921311378479, 0.6811136603355408, 1.0794379711151123, 0.5638790726661682, -0.011651640757918358, -0.13335147500038147, 0.1307699829339981, 0.25657936930656433, 0.7956516742706299, -1.2709627151489258, -0.411792129278183, -2.2470040321350098, 1.0181094408035278, 0.32744714617729187, -0.7211393713951111, -0.024052800610661507, 0.36745357513427734, -1.8959370851516724, -0.6720079183578491, 0.5184776186943054, 0.9396705627441406, 0.36289140582084656, 0.42175906896591187, -0.052231088280677795, -0.6131975650787354, -0.01115088164806366, 1.2719151973724365, -1.7619190216064453, -2.426081418991089, 0.5447575449943542, 0.17784182727336884, -0.08502635359764099, -1.0029464960098267, 1.299827218055725, -0.5417336225509644, -2.6232290267944336, 0.3004065752029419, 0.1991063803434372, -0.25043776631355286, 1.4002323150634766, -1.934404969215393, 0.6391997933387756, 0.5780212879180908, -1.0592931509017944, -1.4503625631332397, -0.36619049310684204, -2.5819478034973145, -0.8945547938346863, 0.5582563281059265, -2.132749319076538, -2.5441818237304688, 1.255312442779541, 0.21462787687778473, 0.9314247369766235, -0.7881275415420532, -0.8167751431465149, 0.08805236965417862, 0.14793555438518524, 0.5091047286987305, 1.5318418741226196, 0.5887099504470825, 0.9403124451637268, 0.4727344810962677, 0.5869165658950806, 0.35189089179039, -0.5036430358886719, -0.75914067029953, 0.11248718202114105, 0.2561788856983185, 2.1119256019592285, -0.12432858347892761, -0.8732360601425171, 0.20163512229919434, -0.24620391428470612, -1.0270700454711914, 0.5397205352783203, 1.7242703437805176, 0.925929605960846, 0.08403661847114563, -0.29578354954719543, -1.0416256189346313, -1.3486493825912476], [0.3978249132633209, -2.085604667663574, -1.0192506313323975, 0.9572039246559143, 0.9612659215927124, -0.15578900277614594, 0.011326813139021397, 0.3396647870540619, 0.5882478952407837, -0.1759023815393448, 0.6448214650154114, -0.21148553490638733, -1.2321445941925049, 0.0952921062707901, 1.0953086614608765, 1.4819939136505127, -2.2596960067749023, 0.7007001042366028, 0.3170101046562195, 0.5579380393028259, -0.8631937503814697, -0.012275072745978832, -0.04259901121258736, -1.2328702211380005, -0.4081290066242218, -0.35677579045295715, -0.9495894312858582, 0.2808454632759094, 0.19046567380428314, -2.465243339538574, 1.2787590026855469, -0.22032281756401062, 0.524005115032196, 1.4703938961029053, -0.8706077933311462, -0.04295482113957405, -0.6062827110290527, 0.10176585614681244, 0.7520389556884766, 0.7112976312637329, -0.992251455783844, -0.12195505946874619, -0.553117573261261, 1.3371391296386719, 1.3292573690414429, 2.738447427749634, 0.08410774171352386, -0.10055005550384521, -1.0223884582519531, -0.8108880519866943, 0.5289409160614014, -0.6329851150512695, 1.027073860168457, -1.162410855293274, 0.35390743613243103, 0.0673360675573349, -0.10714122653007507, 0.07345022261142731, -0.4108486771583557, 0.49015915393829346, 0.5405465960502625, -0.4879515767097473, -0.9156423211097717, -1.3886405229568481, 1.5693061351776123, 2.0730369091033936, 1.0990983247756958, 1.3037348985671997, 0.352814257144928, -1.886784553527832, 0.5561873316764832, -0.9386051893234253, 0.4522238075733185, -0.4967300593852997, 0.6656754612922668, 0.08506587892770767, -0.4161035120487213, -0.7684203386306763, -0.11460905522108078, -0.34348422288894653, -0.13082855939865112, -0.0936308205127716, -1.0917943716049194, -0.024552203714847565, -0.15868355333805084, 1.3958476781845093, 0.4732637107372284, -0.27318108081817627, -0.5414887070655823, -0.20715317130088806, -1.5464969873428345, 0.5918240547180176, 0.28654566407203674, -0.24394874274730682, 1.2239536046981812, 1.6681914329528809, -0.314956933259964, 2.433429479598999, 1.465722918510437, -1.3831822872161865, -2.6915807723999023, 1.1955294609069824, -0.8463924527168274, 1.535791039466858, -0.3389570415019989, 0.3271549940109253, -1.4787027835845947, 0.4530656039714813, -0.188827782869339, 0.3715381324291229, -0.1198626235127449, -0.06385960429906845, -1.5481767654418945, 1.3919649124145508, -0.2717587351799011, -0.23353198170661926, -0.8524179458618164, -0.71608567237854, 1.8712272644042969, 0.19731669127941132, 0.3495485186576843, 2.416577100753784, 2.5593197345733643, -1.6907590627670288, -1.8388986587524414, -1.6170742511749268, 0.5721204876899719, -1.255696415901184, 1.9435648918151855], [0.1986609250307083, 0.4374748170375824, -0.6468058228492737, 1.4704804420471191, -0.30798473954200745, 1.1324658393859863, -0.21573075652122498, -0.015719125047326088, -1.299616813659668, 2.162151575088501, -0.14174573123455048, -0.5234631299972534, 0.03747715801000595, -0.44857361912727356, 0.9378129839897156, 1.9101959466934204, 0.13170744478702545, -1.0148191452026367, 1.2437955141067505, 0.6651321649551392, 0.18734368681907654, 0.08292560279369354, 1.2509924173355103, 1.334660530090332, -0.022044287994503975, 1.2404464483261108, 2.1556618213653564, -0.5974308848381042, -0.962421178817749, -0.040135957300662994, 1.0325201749801636, 1.8688338994979858, 0.3349919617176056, -0.5785401463508606, -0.20244115591049194, -1.6579406261444092, -1.5333019495010376, -0.7047302722930908, -0.3776596188545227, -2.0338189601898193, 0.6283870339393616, 0.9340429306030273, -0.9991689920425415, -0.33548402786254883, -0.8028957843780518, 0.37634822726249695, 1.2658573389053345, -0.36765211820602417, 1.6539465188980103, -2.090137481689453, -0.5876991748809814, 0.9542500972747803, -1.5705530643463135, 0.20711655914783478, -0.7459120154380798, 2.452653169631958, -1.1107430458068848, -0.8513416647911072, -0.9250574111938477, -0.7021613717079163, 0.3072514832019806, -1.3669832944869995, 0.46699923276901245, -0.22654592990875244, -1.6115399599075317, -0.7235826253890991, 0.3245749771595001, -0.2665678858757019, -2.0069968700408936, -0.9911457300186157, -1.3465003967285156, -0.27246347069740295, -0.4598291218280792, -0.27427083253860474, 0.3600161373615265, -0.10148642212152481, 0.10888995230197906, 0.4409931004047394, 0.12021204829216003, 0.8377487659454346, 0.23944704234600067, 0.10583579540252686, -0.07553625106811523, 0.36523762345314026, 0.9861143827438354, -2.356081247329712, -0.5861790180206299, 0.42212140560150146, -0.5633814930915833, -1.2962149381637573, 2.1501519680023193, 1.4188133478164673, 2.124485969543457, -2.417309522628784, -0.7382253408432007, 0.7705450654029846, -0.07857523113489151, -0.8268516659736633, 1.1746015548706055, -1.5667041540145874, -0.8490698337554932, 0.9846497774124146, 0.43895137310028076, -0.31294935941696167, 0.6872091293334961, -0.979418158531189, -0.5153424739837646, 0.8015161156654358, 0.9232053756713867, -0.36973536014556885, -2.0455360412597656, 0.5142793655395508, -0.5927035212516785, 0.8034404516220093, -0.04226631298661232, 0.748712956905365, 0.8483905792236328, 0.3738529086112976, -0.34587562084198, 1.1352190971374512, -0.2698478698730469, -0.4201502799987793, 1.38406503200531, -0.025750160217285156, -1.2137943506240845, 1.1446095705032349, 1.6816344261169434, -0.3575504720211029, -0.831423819065094], [-0.49867045879364014, -0.08242437988519669, -0.5485939383506775, -1.0351946353912354, 0.7831985950469971, -0.610479474067688, 1.1681019067764282, 0.7023059725761414, 0.7299847602844238, 0.23317718505859375, 0.5718567371368408, 0.21215426921844482, -0.3372597098350525, -2.894503593444824, 0.6894853115081787, -0.9285270571708679, 0.19072911143302917, 2.099243640899658, 0.982413649559021, 0.8260424137115479, -1.1539428234100342, 0.05602150410413742, -0.9894114136695862, 1.6437617540359497, 0.8985693454742432, 0.0098385289311409, 1.0687755346298218, -0.518473744392395, 0.08539986610412598, 0.8540760278701782, 0.9988663196563721, -0.31911996006965637, 0.39944785833358765, -0.26229795813560486, -0.21498265862464905, -0.4502897262573242, 0.0664341002702713, -1.2694246768951416, 2.4392905235290527, 0.8648552298545837, -0.028500212356448174, -0.5401004552841187, 0.22638843953609467, -0.005240154918283224, -1.417226791381836, -0.6249983310699463, -1.6588150262832642, 0.8142932653427124, 0.8317518830299377, -0.26623061299324036, 1.303605318069458, -0.7233542799949646, 0.7963669896125793, -1.1210377216339111, 1.07962167263031, 1.5406622886657715, -1.1992501020431519, -0.6102625727653503, 2.3673770427703857, 0.5578386187553406, 0.08809660375118256, -1.1782628297805786, -0.7046535611152649, -2.0642757415771484, -0.047085367143154144, -1.5760343074798584, 0.23330232501029968, 0.20021989941596985, -1.4009283781051636, 2.56174898147583, 0.539726734161377, 0.9134691953659058, 0.5100995898246765, -0.2403847724199295, 0.428551584482193, 1.6105620861053467, 1.108824610710144, 0.25230780243873596, -1.1503639221191406, 0.6747356057167053, 0.41927558183670044, 0.6108906865119934, -1.3854455947875977, 0.8255335688591003, 0.02411821484565735, -1.5617291927337646, 0.3749389052391052, 0.11946886032819748, 1.3287097215652466, 0.7992806434631348, -1.1509299278259277, -0.3724101781845093, -1.6306638717651367, -0.10048651695251465, 0.4855668246746063, 1.247043490409851, -0.09468654543161392, 0.0622166283428669, -1.1692267656326294, -0.9116722345352173, 0.7964012026786804, 0.6055554747581482, -1.6179070472717285, 0.5832343697547913, -0.0007168579031713307, -0.3256351053714752, 0.39165908098220825, -0.5258326530456543, 1.6219496726989746, 2.1749155521392822, 0.6837766766548157, -1.3110135793685913, 0.1106746569275856, -1.1300572156906128, 0.3126501441001892, -1.2382551431655884, 0.3300600051879883, -1.6889392137527466, 1.2253987789154053, 0.8193130493164062, -0.9913200736045837, 0.4228353798389435, 0.876754105091095, -1.5271995067596436, 0.6852880120277405, -1.1236284971237183, -1.7504626512527466, -0.3706369400024414, 0.0010207880986854434], [2.0605547428131104, -1.2293758392333984, -0.4438925087451935, 0.06457522511482239, 0.5774073600769043, 0.8065077662467957, 0.9461845755577087, 0.7641241550445557, -1.0986275672912598, -0.8669962882995605, 0.6906313300132751, -0.5435181856155396, -1.8126112222671509, 0.3367549180984497, 1.2775779962539673, 0.5161153674125671, -0.4753722846508026, -0.5755772590637207, 0.18614241480827332, 0.411287397146225, -0.14276444911956787, -0.4193830192089081, 0.5432431697845459, -1.070397973060608, -0.1536494940519333, 1.0801552534103394, 1.373164176940918, 0.43259158730506897, 1.1373947858810425, -1.9110978841781616, -0.28665852546691895, 1.825010895729065, 0.3097451329231262, 0.18272759020328522, 0.4082196056842804, -0.7693681120872498, 1.0634735822677612, 0.173628568649292, 2.3855299949645996, -1.0555609464645386, -0.1530625969171524, -0.5827690958976746, -0.1285424679517746, 0.4097159206867218, 0.38981303572654724, 1.573643684387207, -1.2672866582870483, -0.5283603668212891, -1.3403998613357544, 0.37667372822761536, 0.43041881918907166, -0.34362104535102844, -0.447388231754303, 1.4806551933288574, 0.47434312105178833, 1.9520161151885986, -0.6647026538848877, -0.2951071262359619, 0.5155681371688843, 0.4494684338569641, 1.1288697719573975, -0.4767034947872162, 2.0715951919555664, -1.7850135564804077, -0.2987133264541626, -0.9907703995704651, -0.2086239457130432, -1.0621726512908936, 0.0034360147546976805, -0.6171733140945435, -0.9821436405181885, 1.3617891073226929, -1.3949235677719116, -0.13233555853366852, -0.1487611085176468, -0.20815850794315338, 0.03961651772260666, -0.20125268399715424, -1.7363530397415161, -0.423859566450119, 0.0042452504858374596, -0.5814450979232788, -1.4061280488967896, -0.02697465755045414, -0.7464185357093811, 0.6470427513122559, -0.2983258366584778, 0.11702000349760056, 0.3340076506137848, -0.7232664227485657, 0.6773767471313477, 0.05928314849734306, -0.23268885910511017, 0.7755393981933594, 1.0662152767181396, 0.04241637513041496, 0.8034626245498657, 2.2129955291748047, 0.4082080125808716, 0.9092822670936584, 0.6119558811187744, 0.4062308073043823, -2.1482808589935303, -0.245309516787529, 2.1173906326293945, -0.5825032591819763, 0.14638008177280426, -1.2691104412078857, -0.1995418816804886, -2.1018688678741455, 0.37967851758003235, 1.9846680164337158, -1.0001884698867798, -0.8321085572242737, -0.08560031652450562, -0.257455438375473, 0.762829601764679, 1.098031759262085, -0.8039485812187195, -0.25683966279029846, 1.7994694709777832, -0.5837170481681824, -0.13994526863098145, 0.3449333608150482, 0.9598183631896973, 0.007114635314792395, -0.9784778356552124, 0.5396201014518738, 0.3247232735157013], [0.026348872110247612, 0.0011448254808783531, 1.1329954862594604, -1.8912992477416992, -0.12836866080760956, -0.31230175495147705, 0.3966929614543915, -0.6553465723991394, -1.7097711563110352, 1.1841747760772705, 0.8317227363586426, 1.0914280414581299, 1.1223353147506714, -1.381978154182434, -0.4634615480899811, -1.215523600578308, -0.16799664497375488, -0.334026575088501, 0.028340250253677368, -1.0634819269180298, 2.397679567337036, 0.1456412374973297, 0.2254822850227356, -0.7930600643157959, -0.7718015909194946, -0.600741982460022, -0.7280776500701904, -0.3166120946407318, 0.7202031016349792, 0.3602559566497803, 1.3747700452804565, 0.6135362386703491, 0.10054843872785568, 0.0574897862970829, -1.3026384115219116, -0.32434308528900146, 1.2579630613327026, 0.04920491203665733, 0.09966170787811279, -0.9267919659614563, 0.0640023946762085, -1.6161011457443237, -0.4766928553581238, -0.04828276112675667, 0.5668279528617859, 0.5064224004745483, -0.8240652084350586, 0.052389346063137054, -1.045711636543274, -1.4716670513153076, 1.062066912651062, 0.6865313649177551, 0.2856951057910919, -1.2595808506011963, -0.5927069187164307, 0.8340622782707214, 0.22328273952007294, 1.3839739561080933, -0.4394437372684479, 0.41128745675086975, -0.09005305916070938, -0.6715235710144043, -1.0170913934707642, -0.8198948502540588, -0.5497898459434509, 0.3331115245819092, -1.6844457387924194, -0.660108208656311, -0.26564928889274597, -1.2800650596618652, 0.6584018468856812, -0.8875695466995239, 0.8289166688919067, -0.09172119945287704, -1.5084987878799438, -0.4984067380428314, -1.2131749391555786, -0.78229820728302, 1.604706048965454, -0.161907359957695, 0.28177782893180847, 0.40600547194480896, -2.409614086151123, 1.0175960063934326, 0.3883708417415619, 0.2511194944381714, 0.19085797667503357, -0.37686777114868164, 0.4529629349708557, 0.447812557220459, 0.3310786485671997, 0.15065525472164154, 2.949169158935547, 1.2809425592422485, 0.5460807085037231, 0.17255347967147827, -0.3429647386074066, -1.310318112373352, -1.1081631183624268, 1.5446434020996094, 0.8694819211959839, 0.9804173111915588, 0.7114336490631104, 0.6204454898834229, -0.6803807616233826, -0.9094432592391968, 0.8713125586509705, -1.4472512006759644, 2.772336006164551, 0.2021452933549881, 0.8950310349464417, -0.21961510181427002, -2.192819833755493, -0.28269389271736145, 0.42909669876098633, 1.2992463111877441, -0.7432097792625427, -0.44807764887809753, 0.52785724401474, -0.3396225571632385, -1.9276072978973389, 2.2054080963134766, -0.6911576986312866, -1.4957103729248047, 2.441518783569336, 0.986336886882782, -0.30724841356277466, 0.28250837326049805, 0.4832683801651001], [-0.10322728008031845, -0.6717937588691711, 2.322671890258789, -1.4835416078567505, -0.953341543674469, -0.26356014609336853, 1.1037036180496216, -0.5586961507797241, -0.4050387442111969, 0.36151134967803955, 0.28863245248794556, 0.9060469269752502, -1.2044074535369873, 0.4648554027080536, 0.34490880370140076, 0.8189239501953125, -1.1229920387268066, 0.8287469148635864, 0.5604900121688843, 0.08614825457334518, -0.9215786457061768, -1.0821113586425781, 0.1055501326918602, 0.8967731595039368, -0.37731820344924927, 1.1369397640228271, 0.8026220202445984, -0.2550913095474243, 0.5680232644081116, -0.6720985174179077, -0.1726311445236206, 0.030607564374804497, -0.671037495136261, -0.027768699452280998, -1.1870434284210205, -0.6073946952819824, -1.0843006372451782, -0.6539857983589172, -0.6527822017669678, -3.1440060138702393, 1.5340113639831543, 0.32186341285705566, 0.8613669872283936, -0.40156304836273193, -1.4307690858840942, 0.22052906453609467, 1.42978036403656, 0.7048252820968628, 0.7286027073860168, -0.03132491931319237, -1.0403987169265747, 0.6170743703842163, -1.0245366096496582, -0.09032367169857025, 0.20202618837356567, 0.6539298295974731, 0.5691144466400146, -0.040798287838697433, 1.0669033527374268, -0.859843373298645, 0.3898719847202301, -1.2317543029785156, -0.3173021376132965, 1.0940051078796387, 0.24764683842658997, 0.6363586783409119, 0.7941676378250122, 0.14981183409690857, -0.45510342717170715, -0.22637119889259338, -0.3414275050163269, 1.8233591318130493, 0.9596654176712036, 0.40989771485328674, -0.8979256749153137, -0.824279248714447, 1.419318437576294, -0.7414339184761047, 0.3308323323726654, 0.782389760017395, -1.127106785774231, -1.0731819868087769, -0.37781956791877747, 0.6854193806648254, 0.1700328290462494, -1.0569871664047241, -0.19061000645160675, 0.16854648292064667, -2.557934284210205, -0.9048068523406982, 0.24276942014694214, -0.18859411776065826, 2.0637078285217285, 1.5744655132293701, -0.49382832646369934, -1.1485928297042847, -0.05180467292666435, -1.0385468006134033, -0.18746820092201233, 1.8137974739074707, -0.8195813894271851, 0.6532180905342102, 2.168944835662842, -0.7317087650299072, -0.3868122100830078, -1.9099459648132324, -1.3490424156188965, 0.7469000816345215, 0.13023880124092102, -0.07211553305387497, -0.9474802613258362, 0.018539099022746086, 0.4127650856971741, 0.2774946987628937, 1.209978699684143, -0.573046088218689, -1.2131361961364746, 1.091177225112915, 0.7116606831550598, -0.34489715099334717, 0.9421283006668091, -0.08389168977737427, -0.21240289509296417, -0.08259100466966629, -1.192989468574524, 0.2341831773519516, 0.6434498429298401, -0.20394639670848846, -0.22596415877342224], [1.3219501972198486, 0.3399757444858551, 0.7811480164527893, -0.8312743902206421, -2.518522262573242, 0.6617142558097839, -0.12725038826465607, 0.6534596085548401, -2.3677022457122803, -0.10090342164039612, -1.0027648210525513, 2.9731526374816895, 1.7496403455734253, -0.19029119610786438, 0.5732086896896362, -1.009633183479309, 1.453490972518921, -0.7122370004653931, 0.7284845113754272, 0.07744377106428146, -1.7090708017349243, -0.6162488460540771, -0.7988042831420898, -0.5842361450195312, 0.427949994802475, 0.5070071816444397, -0.057950954884290695, -0.9480684995651245, 0.32141146063804626, 0.5761789679527283, -0.1754254549741745, 1.2525835037231445, 0.9540269374847412, -0.603893518447876, -0.35563892126083374, -1.2128185033798218, 1.900604009628296, -1.05879545211792, 0.045119497925043106, 1.172343373298645, -0.6955137252807617, 2.017683982849121, 1.7979357242584229, 1.8371762037277222, 0.07533440738916397, -0.9660206437110901, 0.7512512803077698, -0.47740286588668823, -0.06638839095830917, 0.10191505402326584, -1.7507619857788086, 1.084023118019104, 0.46473783254623413, 0.06347987055778503, 0.4785902500152588, -0.6065524220466614, -1.3420593738555908, 1.8764549493789673, 0.028172343969345093, 0.433487206697464, -1.3848443031311035, -0.993826150894165, -1.5599982738494873, -0.997111976146698, -0.6001662611961365, -0.8376402854919434, 1.8577063083648682, 0.9783729910850525, 1.2859808206558228, 0.33650103211402893, 0.7627710103988647, 0.6921126842498779, 1.8221899271011353, -1.7238495349884033, -1.8409488201141357, 0.883737325668335, 1.7010680437088013, -1.186112880706787, 1.558271050453186, 1.4019346237182617, -0.3283853232860565, 0.6923350691795349, 1.59283447265625, 2.365980863571167, -0.28617408871650696, -0.6135856509208679, 0.9416815638542175, -1.0429240465164185, 0.030025633051991463, 0.31941163539886475, 1.2564915418624878, 0.2724757492542267, 0.9826361536979675, 0.2596662640571594, 0.9296586513519287, -0.20317697525024414, 0.465646356344223, -0.4678910970687866, -0.06720954924821854, 0.4960479438304901, 0.2556043267250061, -0.826857328414917, -0.9687883257865906, -0.12182259559631348, 0.7469469308853149, 0.8958078622817993, -0.22997313737869263, 0.26473087072372437, 0.21263550221920013, 0.007855193689465523, 1.1669793128967285, -1.6454685926437378, -1.2024908065795898, 0.45764705538749695, -1.153588891029358, 0.5720864534378052, 0.7815935611724854, -0.6970109939575195, 0.9085288047790527, -0.0036003882996737957, -1.3370400667190552, -2.0472252368927, -1.0010048151016235, -0.3468034267425537, 0.8866720795631409, 0.06076911836862564, 0.2010403424501419, 1.311752200126648, 0.6459248661994934], [-0.5989122986793518, -0.03462135046720505, -0.24842743575572968, -0.10269080102443695, 1.223481297492981, -0.7645983099937439, 0.5117451548576355, -0.32906776666641235, 1.1160961389541626, -0.559531569480896, 0.22076092660427094, -2.1280834674835205, -0.5709955096244812, -0.42143481969833374, -1.6015636920928955, -0.5676566362380981, -0.9392434358596802, 0.7583063840866089, -1.4170124530792236, -0.17405040562152863, 2.3407955169677734, 1.6495394706726074, 0.8729438185691833, -0.021713146939873695, -0.20500653982162476, 0.3428100645542145, 0.1353522688150406, -0.39490795135498047, 0.2575928866863251, 0.15431217849254608, 0.4270769953727722, 1.8218977451324463, -0.10303371399641037, 0.3500879406929016, -0.039959002286195755, 0.4641152024269104, 1.1395567655563354, 0.13665373623371124, 0.7604783773422241, 0.12723344564437866, -0.3890281915664673, 0.17619702219963074, 0.7378599643707275, -0.12648656964302063, 0.11362259089946747, 1.2266920804977417, -0.48569631576538086, 0.9237022995948792, -0.27565833926200867, 1.0151904821395874, 1.6339666843414307, 2.2288899421691895, 1.0903499126434326, 2.6137681007385254, 3.528195381164551, -0.5658787488937378, -0.30329960584640503, 0.851107656955719, 0.06203927844762802, 0.42437922954559326, 0.5729928612709045, 0.050749845802783966, 1.1747955083847046, -0.7736581563949585, 1.3480557203292847, 1.0693392753601074, 0.6330494284629822, -2.0412118434906006, -0.28907495737075806, -0.046133577823638916, -0.1577831506729126, -0.49654659628868103, -0.3503536880016327, 0.4781011939048767, 0.2990557551383972, -0.7527565360069275, -0.9021131992340088, -1.305830955505371, 1.9798418283462524, -0.5966112613677979, 0.08508230000734329, -0.04178896173834801, 1.9110960960388184, -1.1885432004928589, 1.828474998474121, 0.7080705761909485, 0.6656351685523987, -0.0488862618803978, 0.88193678855896, 0.05192144215106964, 0.5162568688392639, -0.3181121349334717, 0.6476874947547913, 0.41801825165748596, 0.7178574800491333, -0.157477468252182, -0.27812376618385315, 0.16762559115886688, 0.20010557770729065, -0.6037737727165222, 0.6077665090560913, 0.716649055480957, -0.1766836792230606, -0.4069647490978241, 0.17442117631435394, 0.1507960706949234, 0.9810782074928284, -0.21555481851100922, 1.3933510780334473, 1.326758861541748, 0.9847838878631592, 0.1797982007265091, 0.06206842511892319, -0.540102481842041, 0.31810081005096436, -0.5092408061027527, 0.23039115965366364, 0.6901925206184387, 1.3883090019226074, -0.6185234189033508, 1.0395584106445312, 0.09838750213384628, 0.4647941291332245, -0.6281063556671143, 0.11348740011453629, -0.5266772508621216, -0.3595427870750427, 0.19135060906410217, 0.17662793397903442], [-0.7275508642196655, 1.2559219598770142, -0.6620354056358337, -0.07983679324388504, 0.9969620704650879, 0.3691481947898865, -0.528763473033905, 0.4197179675102234, -0.20782093703746796, -0.7477504014968872, -0.35182833671569824, 0.7246842384338379, -0.3677252233028412, -1.009575366973877, -1.0392709970474243, -0.06976108998060226, 0.024788083508610725, 0.8136621117591858, 0.5482305884361267, 0.34292373061180115, 0.6013577580451965, 0.3787403404712677, -0.8402647376060486, -1.0828779935836792, -2.101710557937622, 0.6984550356864929, -0.40989333391189575, -0.19513753056526184, 0.4123288094997406, -0.8152481317520142, -0.6833142042160034, -1.311299204826355, -0.47326117753982544, -0.541739821434021, -0.9239758253097534, -1.5301870107650757, -0.2840697467327118, 0.48917731642723083, 0.16713827848434448, -0.8066675662994385, 0.12784335017204285, 1.234511375427246, 0.4932219684123993, -1.0226622819900513, 2.181630849838257, -0.5448548793792725, -1.3468122482299805, 0.3893158435821533, -1.5704752206802368, 0.2963583767414093, -0.13419872522354126, -1.332037091255188, 1.0651992559432983, -1.2747546434402466, 0.21639923751354218, 0.8030312657356262, 0.6859880089759827, -0.03041742742061615, -1.2874984741210938, 0.06676317751407623, -0.724930465221405, 0.9010159373283386, -0.33404406905174255, 0.24558402597904205, -0.1761171817779541, -0.6085402369499207, 0.17872314155101776, -0.3790580630302429, 0.13496752083301544, -1.7577332258224487, 0.7241350412368774, 0.7297915816307068, 0.21666346490383148, -0.2321275919675827, -0.7709859609603882, 1.0519740581512451, 0.7261242270469666, -1.0645254850387573, -0.3162235617637634, 0.07328671216964722, -0.5634264349937439, -0.7961255311965942, 0.8936691880226135, -0.13915200531482697, -1.0985718965530396, 0.09038543701171875, -1.7614115476608276, -0.27767863869667053, -1.019243836402893, -0.21355845034122467, -2.634336233139038, -1.0903910398483276, 1.3797229528427124, -0.9697492718696594, -0.9236202836036682, -0.9445919990539551, -0.21967658400535583, -0.4724884331226349, 0.42114564776420593, -1.8653112649917603, 1.4034686088562012, -2.041095018386841, 0.6110602021217346, 0.3421016037464142, -0.7901917099952698, -0.9734681248664856, 3.0688843727111816, 0.3466573655605316, -0.4862838089466095, -0.023609792813658714, 1.2742588520050049, -0.014197761192917824, 1.0175424814224243, -1.219459891319275, 0.003495032899081707, -1.0655255317687988, 1.4903432130813599, -0.16047298908233643, -0.6630938649177551, -1.4830856323242188, 0.41065558791160583, 0.8840492963790894, 1.4613269567489624, -0.19321657717227936, -0.3918413519859314, 1.02467942237854, 0.4683659076690674, -0.8076244592666626, -0.7207282781600952], [-0.6994538903236389, -1.7545206546783447, 0.5871596932411194, -0.05472651496529579, -2.7977116107940674, 0.6772823333740234, -0.863696813583374, -0.5261122584342957, -1.9757965803146362, -1.3337267637252808, 1.2104018926620483, 0.41285961866378784, -0.23408706486225128, -0.44480156898498535, 0.3659200668334961, -0.7399371862411499, 0.8210052251815796, -1.4043971300125122, -0.7344116568565369, 1.2438491582870483, 0.3439770042896271, -0.3003225028514862, -0.12295330315828323, 0.3091494143009186, 0.28752729296684265, 0.6319040656089783, 0.3080913722515106, 0.46827054023742676, -1.745322823524475, 0.06659888476133347, 0.19905485212802887, 0.8033798336982727, 0.8397661447525024, 0.9500412940979004, 1.570439338684082, -0.9860175251960754, 2.6189498901367188, 0.6033859252929688, 1.6496081352233887, 0.7248752117156982, 0.6522190570831299, 0.7581904530525208, 0.9079560041427612, -1.1314153671264648, 1.6695753335952759, -0.7495184540748596, -0.5233350396156311, -0.9487228393554688, 0.8598636388778687, 0.13454307615756989, -1.610093593597412, 0.11490856111049652, -0.1769573837518692, 0.4710128605365753, 0.1718326210975647, -0.5685294270515442, -2.271008253097534, 0.26373597979545593, -0.16881930828094482, -0.7696597576141357, 0.16750285029411316, -0.2789689898490906, -0.25826600193977356, 1.8527119159698486, 0.35356804728507996, -1.1645028591156006, 0.5156689882278442, -0.21478381752967834, 0.2559417188167572, -0.502802312374115, -0.913623034954071, -0.33774533867836, 0.18370242416858673, -0.5455912947654724, 0.19858594238758087, 0.7955505847930908, -1.1199396848678589, -2.988664388656616, -0.5239399671554565, 1.619334101676941, 0.43798840045928955, -1.1155165433883667, -0.005897910334169865, -0.23137763142585754, -0.09051599353551865, 1.631874918937683, 0.17398005723953247, -1.0444658994674683, -0.8038654923439026, 0.020714299753308296, -0.23039866983890533, -0.9147638082504272, 0.45584964752197266, -0.7256680130958557, 0.5686290264129639, 1.1159918308258057, 1.888703465461731, 0.7176048755645752, -0.1329386830329895, -0.5873690843582153, 0.9643717408180237, -0.14557355642318726, -0.05662762001156807, 0.447338730096817, -1.7963008880615234, 0.20764173567295074, -2.5889806747436523, -1.645323634147644, 0.4088814854621887, -1.1434829235076904, -0.49902576208114624, -0.0704822912812233, 0.49388977885246277, -0.3174773156642914, 0.925040602684021, -0.2600398063659668, 1.398979663848877, 0.03183944895863533, 1.3441612720489502, -0.6028841137886047, -0.6700504422187805, 0.5852906107902527, -0.26391682028770447, 0.20874473452568054, -0.6036932468414307, 0.1281898021697998, 0.5099720358848572, 0.23780213296413422, -1.0403945446014404], [-0.013035746291279793, -0.5890944600105286, -0.9159283638000488, -0.25667548179626465, 0.7128767967224121, 0.596463143825531, 0.9576915502548218, 2.218489646911621, 0.5981760621070862, -0.3600524067878723, 0.139274463057518, -0.8061866164207458, 1.0338215827941895, -0.5757056474685669, 0.21280910074710846, 1.2162195444107056, -0.9022915959358215, -0.987216591835022, -0.6913632750511169, -0.641228973865509, -1.4401025772094727, 1.0207093954086304, -1.0752590894699097, 1.6550142765045166, 0.12503734230995178, -0.7794877886772156, 0.4703671634197235, 1.3367966413497925, 1.4642565250396729, 0.32065290212631226, 0.9538613557815552, -0.09997952729463577, 0.6631315350532532, 0.04554181173443794, -1.125004768371582, -1.1292600631713867, -0.7839868068695068, 0.26722025871276855, 0.8282812833786011, 1.5529029369354248, 1.1937201023101807, 1.2785967588424683, -0.6432225704193115, -0.2658236622810364, -0.44124922156333923, -0.4904114902019501, 0.0825752541422844, 0.6783454418182373, 0.59726482629776, -1.3822228908538818, 0.5222826600074768, -1.1310194730758667, 1.1676300764083862, 1.693143606185913, -0.3704584836959839, -0.8664922118186951, -0.7601479291915894, 1.4378360509872437, -0.9513440132141113, 0.9250804781913757, 1.3438409566879272, 1.0052175521850586, -0.08094865083694458, 1.30251944065094, -1.1217541694641113, 1.2427314519882202, -1.628330945968628, 0.23680783808231354, -0.39009395241737366, -0.3449406027793884, 1.7465165853500366, 0.4721183776855469, 0.7520966529846191, 0.201666921377182, -2.2197563648223877, 1.0204970836639404, 2.292858123779297, -0.02628476917743683, -0.23974820971488953, 0.970952570438385, -0.3145136833190918, -0.7343935966491699, 1.2648663520812988, 1.172130823135376, 0.8672739267349243, 2.082003593444824, -1.2350174188613892, -0.006508256308734417, -0.5841533541679382, 2.107398748397827, -1.1843637228012085, -1.0488885641098022, -0.35601383447647095, -0.7577468156814575, 0.492133229970932, 0.34479185938835144, 1.086945652961731, 0.08256000280380249, -1.1647469997406006, -0.28450629115104675, -0.21266797184944153, 0.8682686686515808, -1.179269552230835, 0.18620170652866364, -0.36542561650276184, -0.8593388199806213, -0.2032942771911621, -1.0879777669906616, -0.31195366382598877, 1.6593279838562012, -0.5213735699653625, 0.9705095887184143, -0.6217657923698425, 0.10224798321723938, -0.14140935242176056, -1.4993746280670166, 1.8566100597381592, -1.425398349761963, 1.0292277336120605, 0.45246753096580505, -0.012574068270623684, -0.1514202356338501, -0.7001279592514038, -1.1497514247894287, 0.27968913316726685, -0.05465857684612274, 1.4126050472259521, 0.5149319171905518, 0.4025868773460388], [1.3788599967956543, 1.024454951286316, -0.8405123353004456, 0.3601066768169403, -1.0711838006973267, 0.40684831142425537, 2.080233097076416, -0.9511368274688721, -0.9268823266029358, 1.505688190460205, -0.05666816607117653, -0.2531525194644928, -0.7064711451530457, 0.2465798407793045, 2.0383098125457764, -2.005267858505249, -0.3412284255027771, -0.4091930687427521, 1.0490498542785645, 0.0658925250172615, -0.17933523654937744, 1.424615502357483, -1.0731083154678345, 1.7015926837921143, 2.2137765884399414, -0.31774333119392395, -1.7591484785079956, -1.1527608633041382, 0.4369608163833618, 0.32442861795425415, 0.47521036863327026, -1.6399809122085571, 0.8707777261734009, 0.957023024559021, -1.936082363128662, 0.06115230917930603, 0.8366450071334839, -0.1454296112060547, 0.2929471731185913, 0.11255728453397751, 0.17467914521694183, 1.1180164813995361, 0.2383909523487091, -0.45186853408813477, -0.5192764401435852, 0.025103751569986343, 0.2615187466144562, -1.5840712785720825, 0.1738121658563614, -0.3688550591468811, -0.32691556215286255, -0.8017359972000122, -0.8715000748634338, -0.24046820402145386, -1.0082274675369263, 0.08980386704206467, 0.2925303280353546, -0.042972058057785034, -0.2961295247077942, 1.2303916215896606, -0.8685669898986816, -1.8300508260726929, 0.7842611074447632, -1.641191005706787, 0.1185179054737091, -0.42035725712776184, 0.3086681067943573, 1.9378585815429688, -0.31202149391174316, -0.023925499990582466, -1.1878079175949097, -0.004295604303479195, -0.5652562975883484, 0.7581820487976074, 0.3602745234966278, -0.1693253517150879, -1.5354210138320923, 0.19871637225151062, -0.3472760021686554, -0.7794328927993774, -0.05856149643659592, 0.4073842465877533, -0.07059196382761002, -0.5522134304046631, -0.0652403011918068, -0.3286030888557434, 0.2525014281272888, -1.3293553590774536, -0.2881357669830322, -1.1073641777038574, 0.9041799902915955, 0.5314177870750427, 1.7473673820495605, -0.19974790513515472, 0.5296657681465149, 0.7068399786949158, -0.5564068555831909, 1.605456829071045, -1.6739096641540527, -1.8116910457611084, 1.3265410661697388, -1.4622364044189453, 0.7175125479698181, -0.5490995645523071, -0.9961214661598206, 0.7450231313705444, 0.5382911562919617, 0.08044685423374176, 0.8649340271949768, 0.7041721940040588, -0.10076498985290527, -0.3861164152622223, 0.03988514095544815, -0.08201475441455841, -2.204209089279175, 2.996288776397705, -0.639938235282898, 0.35403433442115784, -0.650931715965271, 0.5680797100067139, 1.8875641822814941, 0.821930468082428, -0.48719340562820435, -0.42586514353752136, 0.3412451446056366, -0.6222553849220276, -0.1284475177526474, -0.7028165459632874, 0.8209853768348694], [-0.705735445022583, -0.48846980929374695, -0.8552331328392029, 1.794757604598999, 1.1212761402130127, 0.07310555875301361, 0.16274333000183105, 0.40875449776649475, -1.638060450553894, -0.4428825080394745, 0.7945804595947266, 0.6765851378440857, 1.723272442817688, -0.8901602029800415, 0.4615910053253174, -0.023116281256079674, 0.20240682363510132, -1.5397465229034424, -0.9047408699989319, 0.3542061150074005, 0.9568062424659729, 1.7998687028884888, 1.7353489398956299, -0.6790952682495117, -1.6097182035446167, -1.2650452852249146, -0.03817210718989372, -1.4629682302474976, -0.7125477194786072, -1.589777946472168, -1.3801209926605225, -0.09169971942901611, -0.38686487078666687, -0.5627838969230652, -0.1502738893032074, 0.09185940772294998, -0.15340113639831543, -0.3545112609863281, 0.17121180891990662, 1.4904085397720337, -0.7697972655296326, -0.22343796491622925, 1.226722002029419, -1.7605241537094116, 0.4622444212436676, -2.1132020950317383, 0.5835064649581909, 1.6459277868270874, -0.08061408996582031, -0.7251332998275757, -1.5868148803710938, 1.1785238981246948, 1.0486087799072266, 0.26332300901412964, -0.410573810338974, 0.38629668951034546, -0.5250526070594788, 0.16863542795181274, 0.9491119384765625, 0.8721950650215149, -1.4201321601867676, -1.4702966213226318, -0.5008508563041687, -1.0524358749389648, 0.1906464844942093, 2.5084221363067627, -1.4470713138580322, -0.7300105094909668, 1.7740734815597534, 2.6358842849731445, -1.2145633697509766, -1.7973809242248535, 1.5432621240615845, 1.6293131113052368, 0.04795682430267334, -0.3005233407020569, -1.749578833580017, 0.43300339579582214, -0.9048702716827393, 0.9518899917602539, -1.8926544189453125, -0.08820472657680511, 2.3893730640411377, -1.5521297454833984, -0.13511428236961365, -0.8007535934448242, 0.7924091815948486, 0.9817348718643188, 0.05144507437944412, 0.8412948846817017, 1.32634437084198, 1.6165080070495605, 2.8995261192321777, -0.3297629654407501, 0.31342458724975586, 1.4606784582138062, 0.6370736360549927, 2.8769729137420654, -0.6843778491020203, -0.17757369577884674, -0.5171787738800049, -1.6770399808883667, 3.710538864135742, -0.8177390694618225, -0.054477810859680176, -1.87161386013031, -1.5818308591842651, 0.2590863108634949, -0.0423620380461216, -0.7029921412467957, 0.17344579100608826, 0.5152921080589294, -0.1964675486087799, -0.7044325470924377, 0.4967884421348572, -0.4783066511154175, -0.07931716740131378, 0.9194433689117432, -2.6059513092041016, -0.7713964581489563, -0.861243486404419, 0.3871375322341919, -0.8737443685531616, 0.15486198663711548, 1.6392276287078857, 0.49930956959724426, -1.7032170295715332, 0.5847427845001221, 0.4386887550354004], [0.9046655893325806, 0.35689830780029297, 0.20959483087062836, -0.6718801259994507, -0.10548614710569382, 0.15371407568454742, 0.9123018980026245, -0.30126968026161194, -0.44326525926589966, -1.3019887208938599, -1.7212257385253906, -0.7926485538482666, 0.20874658226966858, -0.6784479022026062, 0.9011036157608032, -0.5408504009246826, 0.37395575642585754, -1.2253246307373047, -0.8738725781440735, -0.19513270258903503, 2.308790683746338, -0.5001951456069946, -0.18352176249027252, 0.5600259900093079, -0.3240254521369934, 0.8841363191604614, 0.3650588095188141, -0.15530307590961456, 0.4174138009548187, 0.6679369211196899, -0.9225529432296753, -2.1923112869262695, -1.4647914171218872, 0.5823111534118652, 1.1821799278259277, 1.0352203845977783, 0.4738771319389343, -0.11685574054718018, -0.13094757497310638, -1.5225740671157837, 1.1662101745605469, -0.011557642370462418, 0.16408805549144745, -1.5991114377975464, 0.820010781288147, -0.30206093192100525, -0.12071207165718079, 1.15790593624115, -0.016546297818422318, -0.22731727361679077, 2.893507719039917, 0.784458577632904, -2.131563425064087, 1.5619298219680786, -0.41720229387283325, 1.7688857316970825, -0.7759244441986084, 0.7787835597991943, -0.7216373682022095, -0.38495567440986633, -0.316169798374176, -0.28505808115005493, -0.41247841715812683, 0.8145602941513062, 1.6987613439559937, 0.14482420682907104, 1.2482627630233765, -0.7665876746177673, 0.03987051919102669, 0.932680606842041, 0.434398353099823, 1.3539824485778809, -0.9690456390380859, -0.23342545330524445, -2.0624382495880127, -0.7047930955886841, 0.5017392039299011, -1.0707080364227295, 0.2275490015745163, -1.3874694108963013, 0.18620842695236206, 0.9482168555259705, -0.21849927306175232, -0.4413852393627167, 0.5268194079399109, -0.3744872808456421, 0.21918079257011414, 0.0461936816573143, -0.40949299931526184, -1.1002753973007202, -0.8683092594146729, 0.6629219651222229, -1.4090574979782104, -0.44618454575538635, 0.1988089233636856, 0.1943269670009613, -0.8251298666000366, 0.5744959712028503, 0.22688429057598114, 0.7954035997390747, 2.4603278636932373, 0.36970457434654236, -0.6984387636184692, 0.0618300698697567, -1.4525240659713745, 1.5215412378311157, -1.4218038320541382, -0.9117252826690674, 1.2532944679260254, 1.1312837600708008, -2.133493661880493, 1.955669641494751, 0.4591909646987915, -1.679772138595581, 1.1169004440307617, -0.17628374695777893, -1.6845331192016602, -0.9262997508049011, 0.07082861661911011, -1.1484555006027222, 0.9309532046318054, -0.13534952700138092, 0.3970235586166382, 0.5472405552864075, 1.2853337526321411, 1.0230149030685425, -0.6973513960838318, 1.3235442638397217, 2.077631950378418], [-0.25998416543006897, -0.0428982637822628, 1.4114453792572021, 0.5566380023956299, -0.03578650951385498, -2.692682981491089, -2.5584912300109863, -0.06246097758412361, 0.668287992477417, -0.10725001245737076, 0.15572905540466309, 0.8302318453788757, 1.0178760290145874, -0.3425295352935791, -0.6168617606163025, -2.1521353721618652, -0.7073571085929871, 0.3652813136577606, 0.34420427680015564, 0.8218443989753723, -1.6980071067810059, -0.6313601732254028, 0.8804766535758972, 0.6585215330123901, -1.1774269342422485, -0.17928160727024078, 0.045368775725364685, -0.048455726355314255, 0.7394989132881165, 0.23073559999465942, 0.18843576312065125, -1.4816293716430664, -1.764962077140808, -1.3715484142303467, -1.0119109153747559, 1.4004747867584229, 0.13282477855682373, -0.24928522109985352, -0.860929548740387, 1.523055911064148, 0.0048586297780275345, 0.027533218264579773, -1.2769933938980103, 0.21829397976398468, -0.5377513766288757, 1.884559154510498, -0.31261375546455383, 1.1204228401184082, -0.039546508342027664, 0.3772585391998291, -0.3020550608634949, 0.8182665109634399, -0.43586018681526184, -1.2911983728408813, -0.45572543144226074, 0.10505665093660355, -0.32086193561553955, -0.1422692835330963, -0.5727113485336304, -1.075905203819275, -1.7684518098831177, 1.5661462545394897, -0.6650463938713074, 1.7838317155838013, 0.2094590663909912, 0.5862361788749695, -8.923518907977268e-05, -0.6581212282180786, 1.4306926727294922, -0.28636032342910767, 0.03421873226761818, -0.7958133220672607, -0.6179806590080261, -1.5454211235046387, -1.6706122159957886, -0.948708713054657, 0.22347918152809143, -0.040777046233415604, -0.2103685885667801, 1.3695863485336304, 0.9294173717498779, -1.218705415725708, -0.9832452535629272, 0.38363221287727356, 0.30405789613723755, -0.3433287739753723, -0.49263226985931396, -2.2446177005767822, 0.8890088200569153, 0.4408409595489502, 0.1444660723209381, 1.1306942701339722, -0.885888934135437, -0.33648619055747986, 1.6461275815963745, -0.03888162598013878, 0.07529818266630173, -1.0405781269073486, -0.5476291179656982, -0.33291390538215637, -0.3668893277645111, -1.4791204929351807, -0.1494525521993637, 0.005012860056012869, -1.4079351425170898, -1.2169427871704102, -0.28365659713745117, 0.9241750240325928, -1.1014676094055176, -0.1878618448972702, -3.3500962257385254, 3.2616958618164062, 0.1185450628399849, -0.7992443442344666, 0.32824578881263733, -1.010877013206482, 0.19857771694660187, 0.22771643102169037, 0.8065114617347717, 0.20520368218421936, 0.6782886385917664, 0.5199217796325684, -0.5454255938529968, 0.337399959564209, 0.7174626588821411, -2.679220676422119, -0.9246653318405151, -1.4966646432876587, -0.43278902769088745], [0.6149237751960754, 0.20134826004505157, -0.2879566550254822, 0.3214685618877411, 1.1991791725158691, -0.8328109383583069, 0.4693443477153778, -1.1811174154281616, -1.627261757850647, 1.7239165306091309, 1.6811375617980957, 1.234400987625122, -0.17152340710163116, 1.8153899908065796, 0.4203997850418091, -0.9575220942497253, -1.5938800573349, -0.5802857279777527, 0.3674449026584625, -0.35025328397750854, -1.094521403312683, 1.3382447957992554, -2.2514808177948, -1.0648895502090454, 0.19662396609783173, 0.09558585286140442, 0.7439347505569458, -0.3663320243358612, -0.16579869389533997, 0.6586576104164124, -1.1219502687454224, 0.2010941058397293, -0.06378807872533798, 2.748216152191162, 0.02722869999706745, 1.1706113815307617, -1.0163636207580566, -1.1838359832763672, -0.10214653611183167, -0.9665451049804688, -0.08096804469823837, 0.26353827118873596, 0.4649282991886139, -0.4027518332004547, 0.6031830310821533, 0.6585906744003296, 0.3577444851398468, -0.9622719883918762, 1.6731117963790894, 0.3937576115131378, 0.08933846652507782, -0.32063791155815125, 0.8158127665519714, 0.11928757280111313, -0.8206875920295715, -0.2521132230758667, -0.3138318657875061, -1.2859997749328613, -0.03660503029823303, 0.9177299737930298, -1.2290786504745483, 1.6913530826568604, -1.313070297241211, -0.666850745677948, -1.082770586013794, 0.10087396204471588, 0.08567915111780167, -1.4268336296081543, -0.9113115668296814, 1.3646942377090454, 1.7618883848190308, 0.39536672830581665, -0.5502313375473022, -0.3395783603191376, -0.33917292952537537, -1.2183111906051636, 0.7127596139907837, 1.2329990863800049, -2.333650588989258, -1.1284371614456177, -0.5973376631736755, -0.5012876391410828, -0.25565147399902344, 0.3974244296550751, -0.8834353089332581, 0.11425671726465225, -1.0837721824645996, -1.0141594409942627, 0.27808815240859985, -1.5663658380508423, -1.473824381828308, -0.6320479512214661, -0.43109214305877686, 1.2785742282867432, 0.8375653624534607, 0.6020960807800293, -1.5135449171066284, 1.0801063776016235, -2.2342135906219482, 1.9756361246109009, -1.381263017654419, -0.1448846161365509, 1.678869605064392, 2.5577919483184814, -0.32336851954460144, 1.2288604974746704, 0.7161999940872192, -1.1937549114227295, -0.31771788001060486, 0.5251120328903198, -0.4267806112766266, 0.27061671018600464, 0.2192274034023285, -0.9929152131080627, -0.6164263486862183, 0.11115356534719467, -1.5384763479232788, 0.7328497767448425, -0.5451200008392334, 1.1613737344741821, 0.5884147882461548, -0.09691928327083588, -0.3096446096897125, -2.0535190105438232, 0.919587254524231, 1.0897226333618164, -0.83363938331604, -1.2099632024765015, 0.9470860362052917], [-1.078321099281311, -0.7791738510131836, 0.8143594264984131, -0.639917254447937, -0.5334576368331909, -0.9640277028083801, 1.3284779787063599, -1.0932519435882568, -0.46276506781578064, -0.41858232021331787, 0.15517087280750275, 0.505872368812561, 0.8660128116607666, 1.8762232065200806, 0.3253556787967682, -2.443937301635742, 1.445928931236267, -0.003980905748903751, 0.7318038940429688, 1.5747756958007812, 0.3553479313850403, 0.6244319081306458, -1.4587504863739014, -0.3252997398376465, 0.2298494279384613, -1.836798906326294, -0.39000779390335083, 0.6541461944580078, 0.21902866661548615, -0.640438437461853, 0.5071527361869812, 1.7224045991897583, -0.33313482999801636, -0.3479222357273102, -0.06017034500837326, -0.8674048185348511, 0.2820725440979004, -2.807680130004883, 0.3071454167366028, -0.4532378911972046, -0.4640870988368988, 1.3931554555892944, -0.45699411630630493, 0.008322576992213726, -2.177438259124756, -0.9301562309265137, 0.9001154899597168, -1.1857821941375732, -1.3724982738494873, -1.2232376337051392, -0.3658379912376404, 1.7643804550170898, 0.8366135358810425, -0.6421650648117065, -0.6884222030639648, 0.5253904461860657, -1.0026577711105347, 0.6501295566558838, -0.8540157079696655, 0.034672949463129044, -0.0773744210600853, 0.18338659405708313, 1.510612964630127, -1.843903660774231, -0.3168330490589142, 0.45234403014183044, -0.4106700122356415, -0.7492963671684265, -0.3372189402580261, 1.3324987888336182, -0.5798875689506531, 0.5463547110557556, -1.0373579263687134, -0.5542152523994446, -0.4915081560611725, -0.015231902711093426, -1.656221866607666, 1.5299841165542603, 0.5749405026435852, 1.0445950031280518, -0.24010737240314484, -0.756348729133606, 0.95859295129776, 0.5124998688697815, 0.11361528187990189, -0.23549707233905792, -0.13293421268463135, -0.42565664649009705, -1.965025782585144, 1.0285954475402832, -0.6638941168785095, 1.0077762603759766, 0.39980509877204895, 0.11629904061555862, -0.9624719619750977, 1.3480347394943237, -0.028071554377675056, 0.7805541157722473, 0.30435943603515625, -0.01664271205663681, -0.3144247233867645, -1.7075903415679932, -0.56144779920578, 0.07446163147687912, -0.4654313027858734, -0.6479200720787048, -1.1514791250228882, 0.5799002051353455, -1.4834576845169067, -1.0141887664794922, 0.20020323991775513, 0.4983953535556793, -1.0882865190505981, 1.4938545227050781, -0.3699568510055542, 1.81298828125, 0.2823205292224884, 0.3344011902809143, -0.436962753534317, 3.1094493865966797, 1.2121282815933228, 0.21239352226257324, -0.21850727498531342, -0.43148428201675415, 1.4429937601089478, 0.5195082426071167, 0.1822938472032547, -1.5311962366104126, 0.23081038892269135], [-2.2721657752990723, 0.7035062909126282, -0.5066922903060913, 0.8905391693115234, -2.0600359439849854, -1.0959986448287964, -0.46988368034362793, 0.7308855056762695, 1.5987975597381592, -0.43496498465538025, 1.3293890953063965, -0.6803718209266663, 0.37315189838409424, -0.9902268648147583, 0.17000749707221985, -1.6680097579956055, -0.3421744108200073, 0.30280423164367676, -2.6728267669677734, -0.3842163681983948, 0.023180199787020683, -0.9795973300933838, -0.259335458278656, 0.264078289270401, -0.6382318735122681, -0.5381930470466614, 0.4217516779899597, -1.607915997505188, -0.2964337468147278, -0.4382825195789337, 1.0894134044647217, 1.1443204879760742, 0.9542034864425659, -0.9359146356582642, -0.503364086151123, 0.22388137876987457, -0.5967245697975159, 0.11617585271596909, 0.40255680680274963, -0.36349397897720337, -1.1471068859100342, 0.4202768802642822, -1.314838171005249, -1.3122731447219849, 0.22140079736709595, -0.21944096684455872, 2.0679309368133545, -0.5051743984222412, -0.24879395961761475, -1.0413808822631836, -0.40582695603370667, 1.8077704906463623, 0.7663124799728394, -0.5841518640518188, -0.6537229418754578, -0.5360047817230225, -1.1183066368103027, 0.44842329621315, -1.2228327989578247, -1.897487759590149, 0.1809195727109909, -0.4282020628452301, -0.527625560760498, 1.026810884475708, -0.12372386455535889, -0.18323712050914764, 0.12323255836963654, 0.6752395033836365, 0.6009880900382996, -0.5632210373878479, -0.9859634041786194, -1.7902575731277466, -1.5165035724639893, -0.9265146851539612, -0.3633967638015747, 0.04603264480829239, -0.9416780471801758, -0.4189097583293915, -1.2902123928070068, -0.623756468296051, 1.4712690114974976, -0.8314830660820007, -1.3464275598526, 0.03391321003437042, 1.5670936107635498, 2.7603209018707275, 2.430723190307617, -0.20918789505958557, 0.18139012157917023, -0.7855784893035889, 0.2300388067960739, -1.1671913862228394, -0.9685657620429993, 1.4739608764648438, -0.8676267862319946, -0.31514447927474976, 0.929618239402771, 1.1961548328399658, 0.8089578747749329, 0.603308379650116, -2.534503936767578, 0.5348517894744873, 1.0812733173370361, -0.46111616492271423, 0.28524249792099, -0.025510236620903015, 0.4031238555908203, -0.5084407329559326, -1.5184231996536255, 0.2993963062763214, -1.0837868452072144, -1.4543986320495605, 1.3643580675125122, 1.685238242149353, 0.7403750419616699, 1.4811221361160278, 1.028430461883545, 0.9130768775939941, -0.9660469889640808, 0.6021798253059387, -1.3882428407669067, 2.662154197692871, -1.6897854804992676, -1.0719478130340576, -1.5727590322494507, -1.745391607284546, 0.2444700449705124, 1.3147534132003784, 0.5147175788879395], [0.002982105128467083, 0.1802007406949997, 0.6275264620780945, -1.2475792169570923, -0.10441658645868301, 0.43311500549316406, 3.2396717071533203, 1.7491694688796997, 0.05231009051203728, -0.8706998229026794, 0.810575544834137, 0.3392493426799774, 1.508004903793335, 1.8486182689666748, 0.4386846721172333, -0.9021527767181396, -1.189440131187439, -0.23810824751853943, 0.9469976425170898, -0.39604467153549194, -0.5411239862442017, -0.5340862274169922, 0.3979024887084961, -0.21620585024356842, -0.9087139368057251, 1.747978687286377, -0.21710091829299927, 2.4075767993927, -1.0719465017318726, 0.1159474328160286, 1.0551042556762695, 0.9098024964332581, -1.6796138286590576, -0.0014240272575989366, 1.2177878618240356, -1.8729685544967651, -0.7062214612960815, 0.724814772605896, 1.6076936721801758, 2.0096323490142822, -0.5427958369255066, 0.8673578500747681, 0.07329116761684418, 1.497161865234375, -0.20417653024196625, 1.3604779243469238, -0.9140270352363586, -0.5384947657585144, -0.22786691784858704, -0.27453550696372986, 0.8533328771591187, 0.4685008227825165, 0.35580986738204956, -0.3442525565624237, -0.6715308427810669, -1.0568490028381348, 1.4334852695465088, -1.3586695194244385, -0.1260344386100769, 0.6126500368118286, -0.7184269428253174, 0.5813546776771545, -0.5497624278068542, -0.9769260287284851, 1.954767107963562, 0.7041786313056946, 1.0716661214828491, -0.19761693477630615, 1.9671988487243652, 1.765956163406372, 0.6869475245475769, -1.3571391105651855, -1.5266269445419312, 0.5524090528488159, 1.1268731355667114, -0.36403316259384155, 0.34558436274528503, -0.414040207862854, -0.22503525018692017, 1.3515329360961914, -0.15778978168964386, -0.42774081230163574, 0.12284064292907715, -0.0007668409962207079, -0.07423968613147736, -1.9033724069595337, -0.38663673400878906, -0.4936876893043518, 0.6162715554237366, 2.3072621822357178, 2.2038862705230713, 0.9305808544158936, 0.8019700050354004, -0.6321337819099426, 2.2376632690429688, -0.7809693217277527, 1.1622902154922485, -0.8152961134910583, -0.3198782503604889, 0.025598974898457527, -1.2191710472106934, -0.6719167828559875, 0.16160757839679718, -0.3648139238357544, 1.3534746170043945, -0.05315902829170227, -1.306015968322754, -1.8421156406402588, -0.8341335654258728, 0.5585125088691711, 1.4085006713867188, 0.27701327204704285, -1.4293506145477295, -0.03326856717467308, 0.26442959904670715, -0.365388959646225, 0.86985182762146, -0.0876777246594429, 1.5501848459243774, -0.9280850291252136, 0.8966550827026367, 1.9379030466079712, -1.4452688694000244, -0.6098327040672302, -0.6954734921455383, 0.8690111637115479, 0.003148632124066353, 2.0276927947998047, 0.3715752065181732], [-0.04209553450345993, 1.4042654037475586, 0.6065582036972046, -1.7327457666397095, 0.06201095134019852, -1.2983428239822388, -0.7365046739578247, -1.029102087020874, -0.6309991478919983, -1.2285679578781128, -0.41154372692108154, 0.735943078994751, 1.032362699508667, 1.1071226596832275, 0.493147075176239, 0.041157785803079605, -1.1346458196640015, -0.9151797890663147, 0.3915964663028717, -1.8055521249771118, 0.14806780219078064, -0.7393221855163574, -0.6083037257194519, 0.07185711711645126, 0.9920804500579834, -0.4602396786212921, -1.1034988164901733, -0.5745298862457275, -0.5260118842124939, -0.4058769643306732, 0.7543146014213562, 0.924014687538147, -1.2997783422470093, 0.9623710513114929, -0.2977901101112366, -0.04207694157958031, 0.41391220688819885, 0.4151706099510193, 1.667876124382019, -1.4859960079193115, 0.7762218713760376, -1.5895037651062012, 1.8272082805633545, 0.9001531004905701, -0.5323811173439026, -0.4428730309009552, -0.9649872779846191, 2.309904098510742, -0.4775480628013611, -0.4210183620452881, -0.7288280129432678, -0.21813838183879852, 2.0402162075042725, 1.145279049873352, 0.6475158929824829, -0.7670105695724487, 0.1858227401971817, 0.056575708091259, 0.22707311809062958, 0.06064819544553757, 0.3015454411506653, -1.4747421741485596, -1.9210834503173828, 0.6754418015480042, -1.5063034296035767, 1.6277236938476562, -1.3115159273147583, 0.7719516754150391, 0.23434273898601532, -0.5264604687690735, 0.7634405493736267, -0.0019441156182438135, 0.6011597514152527, 1.9058493375778198, 0.5655637979507446, 0.601513683795929, -0.45583105087280273, 0.5526145100593567, 1.7262605428695679, -0.5820795297622681, -1.2322925329208374, -0.8460714817047119, 0.014055694453418255, -1.8858038187026978, -0.08278781175613403, -0.14643454551696777, 1.2971603870391846, -1.2095973491668701, 1.9518845081329346, -0.25708144903182983, -1.4915522336959839, 0.9210225343704224, -0.6238773465156555, -0.8341960310935974, -1.1085200309753418, -1.1679649353027344, -0.9818335771560669, 0.30179089307785034, -0.6029840707778931, -0.0528494194149971, -0.2855612337589264, -0.6441601514816284, 0.4525529444217682, -0.10514357686042786, -0.2841029763221741, 0.5175983309745789, 0.7212841510772705, -1.497666597366333, 0.29255977272987366, -0.23468375205993652, -0.1934361457824707, 1.646315097808838, 0.2354350984096527, -0.11830107867717743, -0.29570409655570984, -0.9924812912940979, -0.14973261952400208, 0.24884122610092163, -0.3878137767314911, 0.3845035433769226, 1.1130870580673218, 0.6131240129470825, 1.2468531131744385, -1.165613055229187, 0.25578027963638306, 0.093942791223526, -1.3303163051605225, -0.9396233558654785, 0.6933659315109253], [1.620419979095459, -1.483280897140503, -0.40011346340179443, -0.327581524848938, -0.594494640827179, -0.920066773891449, 1.8689180612564087, -2.1931307315826416, -0.4717490077018738, -0.10534312576055527, 0.2956438362598419, -0.7588313221931458, -0.6465526223182678, -0.2006109654903412, 0.8176122903823853, -0.5831442475318909, -1.8453316688537598, -1.0942975282669067, 0.7796489000320435, 1.0298819541931152, 0.2765021324157715, -0.957965612411499, -1.4467815160751343, -0.6342732310295105, -1.8233599662780762, 0.14008024334907532, -0.26174411177635193, -1.3947994709014893, -2.05169939994812, -1.134198546409607, -0.614509105682373, -0.6689323782920837, 0.27226123213768005, -0.4100893437862396, -0.07873227447271347, -0.9672843217849731, -1.618323564529419, -0.5835122466087341, -0.8593427538871765, -0.7733238935470581, -0.46146273612976074, 0.8275089859962463, 1.434254765510559, 1.1396076679229736, 1.9749565124511719, -0.22776585817337036, -1.2425175905227661, -1.550849437713623, -1.69157075881958, -0.03977781906723976, -0.3757983148097992, 1.1996437311172485, -0.5331124067306519, -0.17705681920051575, 0.042230233550071716, -1.5422781705856323, 0.37297090888023376, 0.4464454650878906, 0.28853467106819153, -0.34063729643821716, -0.12684252858161926, -0.04756053537130356, -1.0721322298049927, -1.0482739210128784, -0.6128998398780823, 1.3647518157958984, -0.13427706062793732, 1.7451752424240112, 1.1293429136276245, 0.29381221532821655, -0.26663488149642944, -0.9376612901687622, -0.71243816614151, 1.3242113590240479, -1.3569172620773315, 0.09860706329345703, -0.32027125358581543, -0.34326404333114624, 1.2639682292938232, -0.9224836826324463, 0.11637697368860245, -1.8729618787765503, 2.4881014823913574, 1.4750934839248657, 0.38558268547058105, -0.0948236733675003, 1.1933494806289673, -0.5824717283248901, -0.4056037962436676, -0.8990801572799683, -2.364896297454834, 1.5524921417236328, 0.8392841815948486, -0.03810017928481102, 0.3002491593360901, -0.13242052495479584, 0.35528481006622314, -0.2983134686946869, -1.561237096786499, 1.0566575527191162, -0.10064233094453812, 1.379023790359497, -0.26459482312202454, 0.8523234724998474, -0.5058401823043823, 0.9983155727386475, -1.9558509588241577, -0.17509613931179047, -1.2141578197479248, 0.4870099425315857, 0.3828009068965912, 0.35856449604034424, 0.0546567402780056, -0.6284412741661072, 0.7285236716270447, 1.0612175464630127, 0.6495072841644287, -0.5125711560249329, 2.1081433296203613, 0.3484332263469696, 0.47242680191993713, 0.3976854085922241, -0.6292397379875183, 1.6660646200180054, 1.4183586835861206, -1.8834508657455444, 0.6459311246871948, -0.5399559140205383, -0.8794469237327576], [-0.8933587074279785, 0.1945866495370865, 1.1203142404556274, -0.9069259762763977, -1.2530455589294434, -0.1376536786556244, -1.0363184213638306, -0.5910733342170715, 0.6251010894775391, -0.6324707269668579, 0.909969687461853, 0.7795802354812622, 1.865383505821228, -1.6688506603240967, 1.1256052255630493, 0.21954230964183807, -0.9541909098625183, -1.426248550415039, 0.6468220949172974, -0.36582663655281067, 1.4234763383865356, 0.07118984311819077, -0.8257018327713013, -0.3661590814590454, -0.15070384740829468, -0.10628260672092438, 2.295515298843384, 0.06084015220403671, 0.7686466574668884, 1.0983552932739258, 1.0642340183258057, -1.141338586807251, -0.225983127951622, -0.104361891746521, 1.8437970876693726, -0.35194748640060425, -1.3168091773986816, -1.4356440305709839, 0.39525407552719116, -1.1597005128860474, -1.352795958518982, -0.9090994000434875, 0.519731879234314, -0.9058753252029419, 1.0767409801483154, -0.27327388525009155, 0.25080934166908264, 0.2703026831150055, -0.7634111046791077, -0.25718945264816284, -0.9749841690063477, -0.19783078134059906, -0.5797109603881836, 0.33417809009552, 0.1756809651851654, -0.9577900767326355, -0.9038980603218079, 0.37453508377075195, 1.6617661714553833, -0.1968987137079239, 0.6120882034301758, 0.05506220459938049, -0.8057239651679993, 0.8666603565216064, -0.36899271607398987, -1.3553860187530518, -0.6584218740463257, 0.5328292846679688, -0.40907520055770874, 0.4050619900226593, 2.019251823425293, -0.6781277060508728, 0.02525978907942772, -0.9556615948677063, -1.6704779863357544, 1.1793500185012817, -0.037469785660505295, -1.218326210975647, 0.08576204627752304, 0.46104347705841064, -1.9205604791641235, 0.553037166595459, 0.4226941466331482, 0.7724101543426514, 0.6258761286735535, -0.05135234072804451, 0.10668692737817764, 0.2993673086166382, -0.7109285593032837, -0.9201851487159729, 1.327736496925354, -1.1098076105117798, 0.03291773051023483, 1.0975277423858643, -1.3227157592773438, 0.3262634873390198, -2.1655397415161133, 0.002582995919510722, -0.89922696352005, -0.8062829375267029, 2.1124231815338135, -2.3454477787017822, 0.6028096079826355, 0.8228720426559448, 0.2642640471458435, 0.5023096203804016, -1.1528947353363037, -0.32864391803741455, -0.3610174357891083, -0.3931688964366913, 1.4667106866836548, -1.0751615762710571, -1.4439613819122314, 0.47224822640419006, -0.9332823157310486, -1.1891320943832397, -0.7621063590049744, 0.8635694980621338, 1.6217578649520874, 0.2742152214050293, -0.586739182472229, 0.9994394183158875, -0.30855894088745117, 0.07327068597078323, 0.5375937223434448, -0.7833611369132996, -0.19011396169662476, 1.4619816541671753, 0.7101957201957703], [-0.36285847425460815, 0.29890409111976624, 0.7818799614906311, -0.58957439661026, 0.7060246467590332, 1.0782763957977295, -1.5220575332641602, -0.20340682566165924, 1.2808270454406738, -0.7786871194839478, 0.4971509277820587, 0.3438177704811096, 0.6444839239120483, -0.6007457971572876, 0.08408808708190918, -0.5679157376289368, -0.10867320746183395, 0.04728057608008385, -0.6622215509414673, -1.8629847764968872, 0.7324910163879395, -1.150874137878418, 0.20253771543502808, 0.7749977707862854, -0.30382734537124634, -1.3659266233444214, 0.497011661529541, -0.5783311724662781, -0.5942500829696655, -0.3445688784122467, -0.5652455687522888, 0.9448614120483398, -0.24506860971450806, 0.9632071256637573, -0.22837898135185242, -0.7716448307037354, 0.6856913566589355, 0.368659108877182, -0.9732716083526611, 0.832836925983429, 0.7372276186943054, 0.7408125996589661, 0.35380083322525024, 1.5937199592590332, 1.7125937938690186, 0.07945527881383896, 1.1512807607650757, 0.7569895386695862, 0.9985823035240173, 0.22484742105007172, -0.5453500747680664, 0.26252466440200806, 0.8551167249679565, -0.1316085010766983, 0.9683927893638611, 0.17045338451862335, 0.029938027262687683, -1.0243216753005981, -1.2375682592391968, -1.284735918045044, -0.9151014089584351, 0.3686138987541199, 0.27476125955581665, 1.4512484073638916, 0.24264158308506012, -0.5714504718780518, 0.8167908787727356, -0.27506524324417114, -0.9768809676170349, -0.09390388429164886, -1.9586670398712158, 1.086373209953308, 1.04662024974823, 0.5857954025268555, 0.22568285465240479, -0.867193341255188, -0.06445881724357605, 0.07585124671459198, 0.7072644233703613, 1.2189048528671265, 2.6235547065734863, 0.6813960671424866, 0.6573638916015625, -0.6880212426185608, -0.6379026174545288, -1.1353123188018799, -0.9643296599388123, -0.08017965406179428, 0.3511119782924652, 0.29333001375198364, 0.8337805271148682, -1.572939157485962, -0.7859613299369812, -0.8169744610786438, 0.47081488370895386, 1.2681804895401, -0.9503756761550903, 0.04542260617017746, -1.1855676174163818, -0.4759529232978821, -1.2252236604690552, 0.7478578686714172, -0.5952276587486267, -2.072568655014038, 0.27215880155563354, -0.18760335445404053, 0.3813115656375885, 0.9183641076087952, 0.47259002923965454, -0.255193293094635, -1.3673210144042969, 0.6072704195976257, 0.3574097752571106, -0.06266793608665466, 0.9128438830375671, -0.05639972165226936, -0.09542760998010635, -0.47617363929748535, -0.4752105176448822, -0.7889572381973267, 0.3119882345199585, -0.5955455899238586, 1.1966253519058228, 1.5483922958374023, 0.8516253232955933, 0.4916818141937256, 0.8266448378562927, -1.0635303258895874, 1.0643081665039062], [-0.8149501085281372, -0.1540115475654602, 1.3588980436325073, 1.0317248106002808, -2.1676313877105713, 0.6385990977287292, 1.274105429649353, 0.7509466409683228, -0.17310117185115814, 1.396831750869751, -0.73121178150177, 1.096348524093628, 1.1885759830474854, -2.2721734046936035, -1.2028191089630127, -0.15376229584217072, -2.1236350536346436, -0.4050631523132324, 1.509740948677063, 0.848612368106842, -0.35840731859207153, 0.5337227582931519, 0.9860227704048157, 0.6549201607704163, 0.8319782614707947, -2.2528717517852783, -1.6509966850280762, -1.469164252281189, 0.3623553514480591, -0.948222279548645, 0.09096629917621613, -2.7801780700683594, 0.20889194309711456, 0.5334063172340393, 0.6366864442825317, 0.4147275984287262, -0.2085653394460678, 1.2480038404464722, -0.7553756833076477, 1.227079153060913, -0.7755470871925354, 0.14708808064460754, -0.19374845921993256, -0.06208832934498787, 0.26800885796546936, -0.30311089754104614, 1.2473034858703613, -1.9024162292480469, 0.46714362502098083, 0.022121381014585495, -1.1207937002182007, 1.3952497243881226, 0.7444417476654053, 1.384657382965088, 1.921100378036499, -0.017861807718873024, -1.3556808233261108, -0.1372210532426834, -0.5540758967399597, -1.470371961593628, -1.1681922674179077, 0.035921476781368256, -0.16612350940704346, 1.4858254194259644, 0.4316730499267578, 0.8225908279418945, 0.44491836428642273, 0.28142181038856506, 0.7901553511619568, 0.2653830647468567, -0.679530680179596, -0.6367390751838684, 0.3725321888923645, 1.4799081087112427, 0.393464058637619, 0.3939942717552185, -1.3690581321716309, 0.6079598665237427, -0.15416809916496277, 0.6247078776359558, -0.21893922984600067, 0.8332101702690125, -0.49756285548210144, -0.35557159781455994, -0.9144290089607239, 1.6158558130264282, 1.1731641292572021, 0.14313462376594543, 0.6743297576904297, -0.6378124952316284, 0.17381544411182404, 1.19614577293396, -0.39559561014175415, 0.2805546224117279, -2.97145676612854, 0.5387216806411743, 0.14346258342266083, -0.2130139321088791, 1.712062954902649, -0.18473921716213226, 0.2317608892917633, -0.8124721050262451, 0.7596827745437622, 0.7791042327880859, 0.4744572043418884, -1.3812745809555054, 1.7827627658843994, 0.6399258375167847, 1.3141117095947266, 0.6218580007553101, 1.1740996837615967, -1.162952184677124, 1.031994104385376, -0.19700004160404205, -0.38037464022636414, -1.2674529552459717, -0.7658731341362, 0.19619740545749664, 0.7050957679748535, -0.7778019905090332, 0.16542011499404907, -0.9396581649780273, 0.9214288592338562, -0.9616797566413879, 0.09486129879951477, 1.242004632949829, -2.051929473876953, 0.6090989112854004, 0.5025288462638855], [-1.1519558429718018, -0.9023613333702087, 0.2916570007801056, -1.060315489768982, -0.19513091444969177, -1.7427504062652588, -1.1444275379180908, -0.7109193205833435, -1.004480004310608, 2.9842779636383057, 0.29349735379219055, 1.4911952018737793, -0.6219066977500916, -0.043319061398506165, -3.2842040061950684, 0.09922517836093903, -0.26851633191108704, 0.8548778891563416, -1.1399060487747192, 0.6917919516563416, 0.6235026717185974, -1.6651487350463867, 0.7923193573951721, 1.2554353475570679, -0.1768377721309662, 0.03696277737617493, -0.8045281767845154, 0.7096170783042908, -2.409257411956787, 1.064098596572876, 0.06048692762851715, 0.022424936294555664, 2.104861259460449, 0.43168187141418457, 1.1560845375061035, 1.2290641069412231, -0.372385710477829, 0.8662445545196533, 0.32958102226257324, -0.2670917809009552, 1.0450528860092163, -0.9243910312652588, -0.5223214626312256, 0.5445370674133301, 0.997978150844574, 1.6149450540542603, 0.1480390578508377, -0.9271196126937866, -1.034590244293213, 0.060149941593408585, 1.4573134183883667, -0.2525254487991333, -0.14987938106060028, 0.8942723870277405, -0.12884268164634705, 1.4350714683532715, -0.5622707605361938, 0.06525147706270218, -1.0285314321517944, 1.6183602809906006, -1.3281162977218628, 0.6170939207077026, -1.3242517709732056, -0.6730642914772034, 0.40914908051490784, -1.529054045677185, -0.5537967681884766, -0.9544905424118042, 0.25467291474342346, -0.01944204606115818, 1.0694622993469238, -1.1552908420562744, 0.24996688961982727, -1.104735016822815, 0.9116635918617249, 0.0899105817079544, -1.2696707248687744, -0.8934291005134583, 0.8149288296699524, 1.7536054849624634, 0.18959592282772064, -0.443045973777771, 1.2343084812164307, -1.0137367248535156, 0.217733234167099, 0.5415981411933899, -0.8312540054321289, 2.000127077102661, -0.22231441736221313, 1.0119928121566772, -0.9009914398193359, -0.3261057734489441, -1.0248104333877563, -0.4114280045032501, -0.12045799195766449, 0.06063450872898102, -0.5980039834976196, 0.2782607972621918, -0.21702273190021515, 0.4995228052139282, -0.7658699154853821, -0.12956543266773224, 0.0974375382065773, -0.18063776195049286, 0.9268658757209778, 0.23096147179603577, -1.3752080202102661, -0.720909595489502, -0.8464086651802063, -0.41476601362228394, 0.5453898310661316, 1.5828934907913208, 1.2296662330627441, -1.2973958253860474, 0.02342294156551361, -1.3147351741790771, 0.7193977236747742, -0.31279316544532776, -0.6014693379402161, -0.893958330154419, -0.6148414611816406, -0.38752463459968567, 1.3690134286880493, 0.046614911407232285, -0.21036884188652039, -0.6779764890670776, 1.162775993347168, 1.0709420442581177, 0.30973944067955017], [-2.8350508213043213, -0.11442464590072632, -1.402742862701416, -0.9358944296836853, -1.0013049840927124, 1.9664839506149292, -0.21993601322174072, 0.15687696635723114, -0.558216392993927, 0.19845595955848694, -0.7522323727607727, -0.686003565788269, 0.6917586922645569, -0.4945526421070099, 0.5431116223335266, 0.8192517161369324, 0.24767060577869415, 1.2737178802490234, -1.3177075386047363, -0.2945551872253418, -1.2873423099517822, -0.07787921279668808, -0.012981187552213669, -1.4937548637390137, 0.3112151026725769, 3.17995548248291, -0.32576799392700195, -0.32773277163505554, -0.6745635271072388, 0.5341593623161316, 1.3382266759872437, 0.054888658225536346, -0.10240867733955383, 1.238659381866455, -1.4857171773910522, -0.35090866684913635, -0.1915474385023117, 1.6674011945724487, 0.9404394626617432, 0.6268457770347595, -0.5338170528411865, -0.8139238357543945, -0.13238459825515747, -0.108320452272892, -0.5173370242118835, -0.14313317835330963, -0.24433398246765137, -0.16124212741851807, -1.0622107982635498, -0.9908486008644104, 0.7575294971466064, 1.1628342866897583, -0.7937975525856018, 0.7359067797660828, 0.30583906173706055, -0.24990525841712952, 0.35326337814331055, -0.6325057744979858, 1.6018413305282593, 0.7413536310195923, -0.6938787698745728, -0.0746237263083458, -1.014438271522522, -0.192757710814476, 0.49223774671554565, 0.147405743598938, -0.10879990458488464, 0.8872225284576416, 0.978644609451294, 0.6072604656219482, 0.7452427744865417, 0.6477092504501343, 0.3173404633998871, 2.549438238143921, 0.017440548166632652, 1.1631089448928833, -1.9970539808273315, -0.26927417516708374, -1.7447845935821533, -0.5011472105979919, 0.8457684516906738, 0.036964815109968185, -0.9450432062149048, 0.645045280456543, 1.4407405853271484, -0.9566927552223206, -0.40018677711486816, 1.7551093101501465, -1.037744164466858, -1.0137310028076172, 0.21123623847961426, -0.3509117364883423, -0.7424631118774414, 0.03883262723684311, -0.5238571166992188, -0.9592989087104797, 0.22206220030784607, -1.1526150703430176, -0.8252171277999878, 0.36589232087135315, 0.9001315832138062, -1.8516688346862793, -0.21974439918994904, -0.8260108828544617, -1.0690988302230835, -0.3151678144931793, 0.37318071722984314, -0.07573983818292618, 0.5072554349899292, -0.21983014047145844, -0.360869437456131, -2.462275981903076, 0.9707564115524292, -1.7790452241897583, -0.5818772315979004, -0.11137787997722626, 0.7763171792030334, -1.246001124382019, -0.2370327264070511, 0.7403305172920227, -0.504463255405426, 0.882119357585907, -0.7307602167129517, -0.6800640821456909, 1.993161916732788, 1.0998042821884155, -0.584997296333313, -1.3692097663879395, 0.8270871639251709], [-1.0112382173538208, 0.21563859283924103, 1.0880637168884277, -0.9631633758544922, 1.5515718460083008, 2.3552188873291016, -0.2423093020915985, -0.040485214442014694, -0.01073615625500679, -0.2766382396221161, 0.39889028668403625, -0.6116179823875427, -1.1021455526351929, -0.9607649445533752, -0.6865983605384827, 0.14813865721225739, -0.5376595258712769, 1.1486402750015259, -1.4813947677612305, -0.0006864016759209335, 0.8774309158325195, 0.29281485080718994, 0.5791855454444885, 1.5611011981964111, 1.0765321254730225, -0.14820066094398499, -0.0418747141957283, -0.05988440662622452, 0.12044283747673035, 0.41238483786582947, -2.8400113582611084, 1.5117464065551758, 0.925491452217102, 0.42334049940109253, 0.5324229598045349, -1.0334628820419312, 0.19099299609661102, -0.6881573796272278, -1.346314549446106, -2.529639482498169, -1.0607460737228394, -0.46651923656463623, -0.35954031348228455, 0.5704103112220764, 0.5570027232170105, 1.6987234354019165, -0.17298759520053864, 0.1594543606042862, 0.808212161064148, -0.22556625306606293, -1.452439308166504, -1.6783690452575684, 0.5170992016792297, -0.08265884965658188, 0.4646419286727905, 0.17440246045589447, 1.266237735748291, -0.12435527890920639, -1.4903032779693604, 1.1579431295394897, 0.08610417693853378, 0.6413140892982483, -0.3914985656738281, -2.072381019592285, 1.049663782119751, -1.3750306367874146, -0.2584182322025299, -0.5250405669212341, 0.2811858057975769, 0.5639190077781677, -0.48827797174453735, -1.409767508506775, -0.5782731175422668, 0.7897143959999084, 0.26767122745513916, -1.5660676956176758, 1.1214380264282227, 0.16036611795425415, -0.87742018699646, 0.6752059459686279, 1.153264045715332, 0.9578449130058289, -0.5291641354560852, -1.256461501121521, 1.0870550870895386, 0.6920651793479919, -0.9828653931617737, -0.24230049550533295, -0.8664445877075195, 0.7260056734085083, 0.11078665405511856, -0.056926924735307693, 1.550308346748352, 1.2715506553649902, 2.467245101928711, 1.1651471853256226, 1.2314085960388184, 0.7269637584686279, -1.3054778575897217, 0.05043626204133034, 1.2789283990859985, -1.1461604833602905, 1.075433373451233, 0.31372636556625366, -1.018563151359558, -0.7680220603942871, 2.0295724868774414, -0.03536946326494217, -0.5772810578346252, 0.4252525269985199, 0.6320711374282837, 0.13160894811153412, -0.64090496301651, 0.3554651141166687, 0.4642478823661804, 2.142695188522339, 1.4816930294036865, -0.0326395183801651, 0.6816709041595459, -0.08972140401601791, -0.40119415521621704, 0.8301207423210144, 0.6579483151435852, 0.046478722244501114, 0.23800817131996155, -0.9108856320381165, -0.5998323559761047, 0.16361311078071594, -0.6560686826705933], [-0.9371259808540344, 0.13331514596939087, -0.6640312671661377, 2.4985504150390625, -1.7938439846038818, -0.9528415203094482, 0.5605450868606567, 1.4105172157287598, 1.1749311685562134, -0.311676561832428, 1.1335481405258179, 1.0587037801742554, 0.4826832711696625, 0.44914838671684265, -0.9148346781730652, 1.239839792251587, 0.8133336901664734, 1.3680437803268433, 0.9616068005561829, -0.26784417033195496, -0.5466013550758362, -1.1910409927368164, -0.24126257002353668, 0.834261417388916, -0.8536205291748047, 0.4014711380004883, -0.5814424157142639, -0.8891633152961731, -0.52756267786026, -1.1777175664901733, -0.13033773005008698, 1.309509038925171, 0.1746840626001358, -0.421831876039505, -1.8182134628295898, 0.8070839047431946, 0.7698033452033997, -0.24651677906513214, 0.37769532203674316, -0.1046401709318161, -1.888919472694397, 0.7765849232673645, 1.5870851278305054, -0.3032677471637726, -0.70767742395401, -1.2227498292922974, -0.5631915330886841, 0.23889541625976562, -0.1274835467338562, 1.6733183860778809, -1.2429511547088623, -0.9190467596054077, 0.790500283241272, 0.8601900339126587, -0.9003255367279053, -0.29990455508232117, -0.04556789994239807, 0.20260196924209595, 0.3617780804634094, -4.13276481628418, -1.29600989818573, 1.95852792263031, -0.16869579255580902, -0.42928069829940796, -0.6692703366279602, 0.768873393535614, -0.3944011628627777, -0.5048524737358093, 1.6366045475006104, -0.41726890206336975, 1.886580467224121, -0.35737326741218567, -0.42493802309036255, 1.3367558717727661, -1.6286578178405762, -0.3552582263946533, 0.5732243657112122, -0.02323842979967594, -0.4954182505607605, -0.36607488989830017, 0.13305209577083588, 0.9986949563026428, 0.8770716190338135, 0.750481128692627, -0.19590520858764648, 0.4148910343647003, -0.024694645777344704, -1.4213998317718506, 0.22191846370697021, -1.1667585372924805, -1.1833693981170654, -0.17444013059139252, -1.3451182842254639, -0.675250232219696, -1.3748910427093506, 0.4149091839790344, 0.2476828396320343, -0.2644341289997101, 1.497959017753601, 0.32469868659973145, 0.7540609240531921, -1.1176598072052002, 1.7492995262145996, 0.2797977924346924, 0.05190533399581909, -1.063462495803833, -1.3507598638534546, -0.5142938494682312, 1.5653692483901978, -0.5292243361473083, 0.4726178050041199, -0.22638818621635437, -0.0162750706076622, -1.2461963891983032, 2.2730090618133545, 0.13318748772144318, -0.20669983327388763, -1.4075359106063843, -0.8282764554023743, 0.6568611264228821, -0.33399277925491333, -1.0757502317428589, 1.8216756582260132, -0.6744337677955627, 0.6094964742660522, 1.6140880584716797, 0.784912109375, -0.2367980033159256, 0.13325877487659454], [1.8085517883300781, 0.8712486028671265, -0.5449360609054565, -1.5472402572631836, 0.6736876368522644, -0.3553394675254822, -0.058694276958703995, -0.6785284280776978, 1.1021082401275635, -1.7969050407409668, 1.65815269947052, -1.1989953517913818, 0.06160278618335724, -0.1409270167350769, -0.7490072846412659, -1.428131103515625, 0.20302090048789978, 0.5534095168113708, 0.49850401282310486, 0.23001086711883545, -0.5770214796066284, 1.8087437152862549, 0.18266022205352783, -0.6542637944221497, -0.1597997099161148, 0.6980029940605164, 0.5244794487953186, -0.9422284960746765, -0.6741717457771301, 0.10245349258184433, 1.0616886615753174, -0.3834872841835022, -2.4962785243988037, -0.698962390422821, 0.08928066492080688, -0.20730049908161163, -0.02862262725830078, -2.0906078815460205, 0.6530815362930298, 0.9580994248390198, 0.33243247866630554, 0.08673088252544403, 0.6074759364128113, 0.33953857421875, -0.331225723028183, 0.11035621911287308, 1.2082431316375732, -0.7627068161964417, -0.7668073773384094, -1.370477557182312, -1.6806433200836182, -0.16822443902492523, 0.33275631070137024, 0.7267733812332153, -0.5508198738098145, -1.2531975507736206, 1.1390628814697266, 0.1450655311346054, -0.17591582238674164, 1.1470414400100708, 0.6734037399291992, -0.38712313771247864, 0.25054627656936646, 1.0026487112045288, 0.5806370377540588, 0.3998979926109314, 1.6023858785629272, -0.6366514563560486, -1.2303701639175415, -0.3979959189891815, -1.5668209791183472, 0.8013020753860474, -2.0609967708587646, -0.49201175570487976, 1.2786774635314941, 0.3736859858036041, -0.15836955606937408, 0.7740728855133057, -0.3699868321418762, -0.44372129440307617, -0.10401485115289688, 0.4452596604824066, -0.9679300785064697, 0.23602384328842163, -0.1745564490556717, 0.21950487792491913, -0.889326810836792, -0.6846576929092407, -0.5702787637710571, 1.463866949081421, 1.5288654565811157, 0.1281653195619583, 0.47197139263153076, 0.9610567688941956, 1.6768383979797363, -1.9207531213760376, 0.9494912624359131, 0.3332503139972687, 2.0244572162628174, -0.2538505792617798, 0.013678343966603279, -1.17317533493042, 0.21346798539161682, 1.8209319114685059, -1.4087235927581787, 0.5530213117599487, -0.2515854835510254, -1.7142894268035889, 0.013018777593970299, -0.2477784901857376, -1.7140064239501953, 0.6414971947669983, -1.0527067184448242, -0.6315141320228577, -0.3851294219493866, 1.230197548866272, -0.07907389104366302, 0.7388516068458557, -0.023383384570479393, 1.2511229515075684, 1.4672422409057617, 0.974055826663971, 1.327330470085144, 0.30660322308540344, -0.7061319947242737, -1.0518012046813965, -1.1517778635025024, 0.9291329383850098, 1.1428258419036865], [-0.1230020672082901, -1.1360986232757568, 2.6808972358703613, 0.04249303415417671, 1.5504097938537598, -1.1410776376724243, -1.3015694618225098, 0.3822462260723114, 0.7362577319145203, -2.94150710105896, -0.8264603018760681, -0.9640688896179199, -1.1570886373519897, 2.332221508026123, 1.7733017206192017, 0.9032610058784485, 1.8257511854171753, -0.5696679949760437, 0.4117918014526367, 0.8045473694801331, 0.8128126263618469, -0.7344286441802979, -1.769800066947937, 0.508173406124115, -0.5317912101745605, 0.8134768009185791, -2.1608853340148926, 1.3318862915039062, -0.42333003878593445, -1.4558907747268677, 0.7298616170883179, -0.5627816319465637, -1.8104296922683716, 0.2747095227241516, 0.3430613577365875, 0.15157686173915863, -0.912655234336853, -0.9952012896537781, 1.4698289632797241, 1.7094357013702393, 0.4130937159061432, -1.660273551940918, 0.8443750143051147, 0.8549537658691406, 0.1603129804134369, -0.26717978715896606, 2.4354939460754395, -0.6672248244285583, 0.8974242806434631, 0.7379676699638367, 0.3079335689544678, 0.28920602798461914, -0.9746100902557373, 0.5058838129043579, -1.1238155364990234, -1.8678375482559204, -0.13284310698509216, 1.6599946022033691, 0.5666606426239014, -0.9071015119552612, -0.08815115690231323, 0.16750890016555786, -0.9529544115066528, 1.7072844505310059, -0.28311315178871155, -1.1950167417526245, 0.29964467883110046, 0.8955946564674377, 1.0571082830429077, 0.2150721698999405, 0.12887635827064514, 0.5522242188453674, 0.18122358620166779, -0.786203145980835, 0.07692918181419373, 0.9072220325469971, -0.9564094543457031, -0.9596989750862122, -0.86358243227005, -0.3032062351703644, -0.9341248273849487, -0.62796950340271, -1.4822818040847778, 0.13816452026367188, -0.6201052069664001, -0.33813995122909546, 1.058099627494812, -0.19242635369300842, 0.14966027438640594, 0.03586770221590996, -2.0224854946136475, -0.024289246648550034, -0.8531901240348816, 0.5392859578132629, -0.31451860070228577, -1.974222183227539, -1.1193163394927979, -0.41948387026786804, -0.057470377534627914, 0.6308490037918091, -0.036894816905260086, -0.03160497546195984, -1.071473240852356, -1.1494371891021729, 0.49438393115997314, -0.8268353939056396, 0.10479891300201416, -0.2114136666059494, -0.6079742312431335, -0.659785807132721, 0.6828110218048096, 0.20302985608577728, -0.3199410140514374, -0.8361298441886902, 0.4732067286968231, -1.3390642404556274, 0.32892271876335144, 1.9957164525985718, -1.1964863538742065, -0.8772866129875183, 0.30021774768829346, 0.8155198097229004, -2.410506248474121, 0.1308089792728424, -1.3031983375549316, 1.2471269369125366, -1.0380275249481201, -2.717893600463867, -0.7158452868461609], [-0.547603964805603, -0.51304030418396, -0.5358035564422607, 0.44736748933792114, 0.490749716758728, 0.3409560024738312, 0.7887740731239319, 0.8492866158485413, 2.0152335166931152, -1.3601988554000854, 0.05386859551072121, 0.6318663954734802, -0.050372909754514694, 0.7441538572311401, -0.7097654938697815, -1.746174931526184, -0.7451115250587463, 1.4146348237991333, 0.6658702492713928, 0.6162069439888, -1.4361943006515503, -0.6354212760925293, 1.3664588928222656, 0.05001834034919739, 0.10468131303787231, 0.774476170539856, 0.39843615889549255, -1.9457381963729858, -2.04707932472229, -1.0795912742614746, -1.15684974193573, 0.0980926975607872, -1.6357232332229614, -0.8499540090560913, 0.9115311503410339, -1.190384030342102, -1.5042084455490112, -0.9720829725265503, 0.685007631778717, -0.2233409285545349, 1.1119710206985474, 2.4847114086151123, 0.3018132150173187, -0.9122433066368103, -0.6736270189285278, -0.23484700918197632, 2.3207333087921143, -1.503686785697937, 0.4523155391216278, 0.05341373756527901, 2.836268186569214, 0.9658923745155334, 0.7032666802406311, -1.1091207265853882, -0.6441895365715027, 0.17954953014850616, -0.32161495089530945, 1.330089807510376, 0.02614949457347393, 1.2249407768249512, -1.5348936319351196, 0.344011515378952, -0.10187476128339767, 0.411604106426239, 0.4186922013759613, 0.5697509050369263, -0.25205758213996887, -0.3691484034061432, 0.9670834541320801, 0.9543622136116028, -1.1295082569122314, 0.44945475459098816, -0.40220898389816284, -0.37970197200775146, 0.9466659426689148, 0.8934722542762756, -1.5051034688949585, -1.086281418800354, 1.431538701057434, 0.17238041758537292, -0.9875657558441162, 0.8900338411331177, -0.11163938045501709, 1.1049317121505737, -0.14216437935829163, -0.4157194197177887, -0.10172393918037415, 0.7489092946052551, -0.11273031681776047, 1.215274453163147, -0.573211669921875, -0.8492319583892822, -0.6233572959899902, -0.11244668066501617, -0.5756062269210815, -0.7875893712043762, 0.9737240076065063, -1.8511766195297241, -0.45691782236099243, -0.773632824420929, -0.6911724209785461, 0.9430800676345825, -1.038835048675537, 1.8279131650924683, -0.3892066478729248, 0.3577023148536682, 0.6305868029594421, -2.2334201335906982, -0.9901211857795715, -0.4872157871723175, 1.0456804037094116, -0.18455980718135834, -1.5133461952209473, 1.0297061204910278, 0.26428866386413574, -0.17902672290802002, 0.47745010256767273, -0.18225491046905518, 1.1712501049041748, -2.2164368629455566, -0.27056312561035156, 0.9752110242843628, 0.12369219213724136, -0.3470061421394348, 0.8484053611755371, 0.639022707939148, -0.21411240100860596, 0.3720182478427887, 0.49969151616096497], [-0.008926810696721077, -0.06961964070796967, 0.10350746661424637, 0.184994176030159, 1.6235904693603516, -1.3709561824798584, -1.1572192907333374, 0.9574782252311707, -0.8308155536651611, -1.1661769151687622, -0.7079045176506042, 1.5176011323928833, 0.9964761734008789, 1.3224151134490967, 1.0887339115142822, 0.5195915699005127, 0.30964937806129456, 1.5475585460662842, 0.8890905380249023, 1.0781830549240112, 0.8776402473449707, 0.3982257544994354, 0.7658811211585999, 0.5641917586326599, -2.351439952850342, -1.076931118965149, 2.1219446659088135, 0.23896166682243347, -1.6167939901351929, 0.3340076506137848, 0.38286083936691284, 1.583696722984314, -1.7648513317108154, 0.2300850749015808, 1.5670063495635986, -0.26251715421676636, -0.06925253570079803, -0.690528154373169, 0.2105419933795929, 0.26603731513023376, -1.1082181930541992, 0.8243898153305054, -0.1403825283050537, 0.8890330195426941, -0.19718001782894135, 0.333233118057251, 0.7817546725273132, 1.249794840812683, 1.333652377128601, -0.47297751903533936, 2.1861040592193604, -0.29179510474205017, 1.6479686498641968, -0.32576918601989746, 0.6276551485061646, -0.9323678612709045, -0.21481259167194366, -0.7423155307769775, -1.0891177654266357, -0.21312353014945984, 0.8891664147377014, -0.9935029149055481, 0.7524755001068115, -1.651711344718933, 0.09615916758775711, -1.4121344089508057, -0.07395800203084946, 2.0690760612487793, 1.05099618434906, -1.3807927370071411, 0.4682377874851227, 1.4365626573562622, -0.8252809643745422, -1.0690782070159912, 1.738726258277893, 1.3874773979187012, 0.16374312341213226, -0.6767616271972656, 0.9224858283996582, -1.0287145376205444, 0.2277103066444397, -2.3983397483825684, 1.052280068397522, 1.6145169734954834, 0.3197147846221924, -0.9910752773284912, -0.9179026484489441, -0.06952245533466339, -0.342171311378479, 0.33669528365135193, 1.4529341459274292, 1.3096007108688354, 0.7313311696052551, 0.13216045498847961, -0.7006530165672302, 0.14038050174713135, 0.6261698603630066, 0.6372324824333191, 0.81032794713974, 1.3399676084518433, 0.10038364678621292, 0.7490744590759277, 1.693777084350586, -0.8861223459243774, -1.503198504447937, -0.41186484694480896, 0.2739879786968231, 0.6470528244972229, -0.22789210081100464, -2.5844597816467285, -0.42739248275756836, -0.43079403042793274, 0.9163327217102051, -0.6803593635559082, -1.9040957689285278, -0.7141314148902893, -0.09629995375871658, 0.6493659019470215, -1.6755189895629883, -1.3380824327468872, -0.9773095846176147, -0.5311196446418762, 0.818450927734375, -0.15153932571411133, 1.2357838153839111, -1.2055275440216064, 1.2055325508117676, 0.43535980582237244, 0.5890054702758789], [-0.29226154088974, 0.7674492597579956, -0.44884973764419556, -0.5378857254981995, 0.1713799238204956, 0.25573238730430603, 0.6871678233146667, 1.4175070524215698, -1.1145471334457397, 0.575615406036377, 0.15191157162189484, -1.3943374156951904, 3.0200552940368652, 0.34678009152412415, -0.7204932570457458, 0.50258469581604, -1.2371976375579834, -2.180598497390747, -0.241623193025589, -1.0106559991836548, -0.12918303906917572, 0.856083869934082, -0.15728513896465302, 1.4266276359558105, -0.14328640699386597, 0.10276221483945847, -0.07192981243133545, 0.6037358045578003, -0.0643867701292038, -1.5058350563049316, -0.13755996525287628, 1.4769833087921143, -0.3191419243812561, -1.804330587387085, -0.26517486572265625, -1.5514956712722778, -0.6611617207527161, -1.9549920558929443, 0.0774335041642189, 0.6382003426551819, -1.3441624641418457, -0.1435004621744156, 0.7612012624740601, 0.021922318264842033, -0.10257864743471146, -0.8221977949142456, -0.5297344326972961, -0.6331846117973328, -0.537345826625824, -0.2129330337047577, -0.30113309621810913, -0.23859350383281708, -1.9933072328567505, -1.0352869033813477, -0.4728340208530426, -0.6545522212982178, 1.1124836206436157, -0.6003095507621765, -0.8655485510826111, -0.6496936678886414, 1.7955189943313599, -1.850786566734314, 0.21549557149410248, -0.34442442655563354, -0.1130470335483551, -0.26307547092437744, 0.16367660462856293, -0.8609113097190857, -1.223057508468628, -1.3719843626022339, -2.5037291049957275, 0.10525570064783096, -3.0535709857940674, -1.6162114143371582, 0.9452144503593445, 2.0994787216186523, 0.858352780342102, 0.013245255686342716, -0.07725050300359726, 1.0120139122009277, -0.15981628000736237, 2.1192033290863037, 0.046618204563856125, -1.230878472328186, -1.3739161491394043, -0.16838443279266357, 0.968561589717865, -0.1720944344997406, -0.5398913621902466, 0.2599773406982422, 0.5115558505058289, -0.2426212877035141, -0.6674994230270386, -0.05960889905691147, 0.5325515270233154, 1.8115615844726562, -0.496448278427124, -0.7848895788192749, 1.9018882513046265, -0.7000121474266052, -0.061619725078344345, -0.8007112145423889, -0.6179569363594055, 1.6667859554290771, -0.44597843289375305, -1.863716721534729, -2.0424649715423584, 0.9191566705703735, -0.5992431640625, 0.685680091381073, -0.06754928082227707, 1.205183744430542, 0.07122313976287842, 0.040795404464006424, -0.12871815264225006, 0.9723177552223206, -0.7344779968261719, 1.3355004787445068, 0.5171215534210205, 0.8264040350914001, 0.7071706056594849, -0.23843161761760712, 1.1020005941390991, -0.9750930070877075, -0.5641733407974243, -1.3943372964859009, -0.18862080574035645, 1.4957270622253418, 0.994023859500885], [-0.5977185964584351, 1.6773285865783691, -0.28233543038368225, 0.09960901737213135, -0.36126548051834106, -1.031079888343811, -0.621497392654419, -0.8425712585449219, -1.0856019258499146, 1.0844876766204834, -0.5753529071807861, -0.5078239440917969, -1.3116331100463867, 1.2215237617492676, -0.13658125698566437, -0.16203656792640686, 0.19420236349105835, -1.0808486938476562, 1.2412678003311157, -0.2586686909198761, 0.8800941109657288, 0.032685939222574234, -1.1559416055679321, 0.2174188494682312, 0.25990673899650574, -1.1006799936294556, -0.40396037697792053, -1.4059845209121704, 0.32443222403526306, 1.128443956375122, -1.3301373720169067, -2.5207762718200684, 0.7325766682624817, 1.0503724813461304, 0.8262925744056702, 0.14207205176353455, 2.117619752883911, -0.44074615836143494, -1.996492624282837, -1.5022332668304443, 0.6340091824531555, -1.7398245334625244, 2.41107439994812, -0.16189239919185638, -0.5938079357147217, -0.15688443183898926, -0.4792501628398895, -0.4633115828037262, 0.643196165561676, 0.17508894205093384, -0.644202709197998, 0.048978425562381744, -0.7526342868804932, -0.5108051896095276, -0.15692971646785736, -0.23898597061634064, -0.415712833404541, 0.21054723858833313, -0.2816866934299469, 1.5844464302062988, -0.4257289469242096, 0.6990795135498047, 0.9248427748680115, -0.7771204113960266, -0.6361693739891052, -0.665494441986084, 0.30226558446884155, 0.22749096155166626, -0.1465091109275818, -0.5112727284431458, -0.09797489643096924, -0.1250378042459488, 1.3244386911392212, -1.7215009927749634, 0.49995431303977966, 1.3566505908966064, 0.14668501913547516, 0.28710976243019104, 0.35787254571914673, 0.4997653365135193, 0.753089427947998, -0.49721479415893555, 1.222652554512024, 0.4007994830608368, 1.030988335609436, -1.466650128364563, 0.18396086990833282, 0.16567352414131165, 1.155380129814148, 0.5789452195167542, 0.19156235456466675, 1.6847180128097534, -1.4443336725234985, 1.2045656442642212, -0.9631338119506836, -0.42710810899734497, 0.3387790024280548, 0.6736705303192139, -0.026310620829463005, 0.44293996691703796, -1.3899078369140625, 1.645359754562378, 0.5843644738197327, 0.40717798471450806, -2.226321220397949, 0.35977068543434143, 0.4043818712234497, 0.7456926703453064, 0.9398099184036255, -0.02636735513806343, -1.0343326330184937, 0.8059530258178711, -1.4103360176086426, -0.3433697819709778, -1.1596697568893433, 1.193796992301941, -0.7490231990814209, 0.708446741104126, 1.488068699836731, -0.781972348690033, -0.02978835441172123, -0.12556830048561096, 1.6318737268447876, 1.8591818809509277, 1.2846382856369019, -2.822479009628296, 1.1129682064056396, 0.9233571290969849, -0.03535495698451996], [-0.37147465348243713, -0.7682892084121704, 1.1828497648239136, 1.1404507160186768, -0.13061636686325073, -0.9004991054534912, -1.1621806621551514, -0.5091298222541809, 1.1111185550689697, -1.095254898071289, 0.16829389333724976, 0.864567756652832, -0.04312281310558319, 1.6936973333358765, -0.15644396841526031, -0.991363525390625, 1.3335909843444824, 0.3453940451145172, -0.34907960891723633, -1.4886369705200195, -0.0027895078528672457, -0.937159538269043, -0.4521186649799347, -0.16421100497245789, -0.4157482087612152, 1.0984694957733154, -1.0739392042160034, -0.192295104265213, -0.10590429604053497, 0.66664719581604, 0.21577076613903046, -0.2420535385608673, 0.06882806867361069, -0.325366735458374, -2.4123353958129883, 0.5823189616203308, -0.6656726598739624, -0.5772913098335266, -1.4471700191497803, 0.5617884993553162, -0.522726833820343, -0.954094409942627, -1.372808814048767, 0.4905926585197449, -1.0079792737960815, -0.4993687570095062, 0.5630143880844116, -2.972383975982666, 0.2958034574985504, 0.6007055044174194, -0.6308714747428894, 1.1450103521347046, 1.323650598526001, -0.2902320325374603, -0.7562094926834106, -1.689883828163147, -0.1428331434726715, -0.33147725462913513, 0.7301444411277771, -0.43316176533699036, -1.3671228885650635, -0.29934942722320557, 0.7433738112449646, -0.5097951889038086, -0.5229693055152893, -0.2847226858139038, 1.5777111053466797, -0.2139693796634674, -0.3549008369445801, -1.4586318731307983, 0.8416077494621277, -0.2954521179199219, 0.33325985074043274, -0.7484372854232788, 0.7374391555786133, 1.829440951347351, 2.055191993713379, 0.41115930676460266, -0.3905029594898224, -2.0554327964782715, -1.6145704984664917, -0.6275701522827148, 0.4653918445110321, 0.14688706398010254, 0.15940941870212555, 0.9213659167289734, -0.9595757126808167, -1.647567868232727, 0.6867485642433167, -1.4414163827896118, -0.5864161252975464, 0.2290642261505127, 0.0030924456659704447, 0.4990324079990387, 1.651546597480774, -0.20693284273147583, -1.0489197969436646, 0.7747635245323181, -1.5453163385391235, -0.4985751807689667, -0.48247087001800537, 0.8515391945838928, -1.3936649560928345, -1.0851060152053833, -1.00434148311615, -0.7427534461021423, 0.23745591938495636, -1.0668461322784424, 0.5159627199172974, 0.13741180300712585, 0.8727437853813171, 0.9101088047027588, -0.9709096550941467, -0.3090803921222687, -0.3182760179042816, -0.14526808261871338, 0.10232153534889221, 1.0124433040618896, 0.6292360424995422, -0.81975257396698, -0.6838362812995911, -0.20759254693984985, -0.02530188299715519, -1.0511291027069092, 0.4953099489212036, -0.6822690963745117, 0.7897799611091614, -1.2976080179214478, -0.7872020602226257], [0.0936683714389801, 2.000704526901245, 0.3217526376247406, -0.8402037620544434, -0.739766001701355, -1.8336857557296753, 2.2932238578796387, -0.251172810792923, -0.5600118041038513, -1.0258880853652954, 0.026579802855849266, -1.3834775686264038, 0.40082305669784546, 0.23482519388198853, -0.1903296262025833, -0.3453449308872223, -1.0903570652008057, -0.5422372221946716, -0.4799213409423828, -0.14277510344982147, 0.33346182107925415, 0.9144993424415588, 1.256139874458313, -0.5024579763412476, -0.19545957446098328, 1.5697498321533203, -1.0350697040557861, 0.17648273706436157, -1.1672829389572144, -0.32312268018722534, -2.0134692192077637, -1.0067667961120605, 0.5557123422622681, 0.7598643898963928, -0.8905760645866394, 0.08202748000621796, -2.2144856452941895, -1.1812032461166382, -0.1331433802843094, -0.5409265160560608, 1.1203328371047974, 0.7422523498535156, 0.911451518535614, -1.7889045476913452, 0.44194692373275757, 0.743013322353363, 1.298348069190979, 1.9265904426574707, 0.6823291778564453, -0.8039082884788513, 0.9572057127952576, 0.66134113073349, 1.3437004089355469, -0.2960592806339264, 0.4061891734600067, 1.706821322441101, -0.014107382856309414, 1.4437834024429321, -1.2028312683105469, -0.47468429803848267, -0.6472262740135193, 0.6997918486595154, 0.6843165159225464, 0.753186047077179, -0.7657265067100525, 1.6096984148025513, 0.013916783966124058, 0.7348975539207458, -1.0180649757385254, 1.6201887130737305, 1.428152322769165, 0.4633837044239044, 0.4020634591579437, 0.10184575617313385, 0.3272707760334015, -0.8726308941841125, -0.9990066885948181, 0.638128936290741, -1.5067849159240723, -1.322943925857544, 0.217585951089859, 0.3274748623371124, 1.9171942472457886, -0.3355848491191864, 0.5745252966880798, 0.08857811242341995, 0.3071858286857605, -0.8818218111991882, 0.23219619691371918, -0.20382337272167206, -0.3416639268398285, 0.5556912422180176, 1.8956512212753296, 0.3361522853374481, -0.5909774303436279, 1.0899364948272705, -1.5860189199447632, -0.3498052954673767, -0.41288015246391296, 1.8153908252716064, -2.236516237258911, -0.5466802716255188, 0.2897733747959137, -0.4503574073314667, -0.02304849959909916, 0.12477755546569824, 0.6696435809135437, 0.9785289168357849, 0.2763427793979645, -0.24878130853176117, -0.4386798143386841, 0.5149697065353394, 0.329485684633255, 0.15989752113819122, 0.4070497155189514, 0.39971473813056946, 0.6819970607757568, 0.5646838545799255, 1.518893837928772, -0.09534092992544174, -1.171528935432434, 0.5272062420845032, 0.9453700184822083, -0.2599570155143738, 0.3438027501106262, 0.20386505126953125, 0.45403853058815, 1.5978816747665405, -0.16266560554504395], [-0.18625129759311676, 0.5428532958030701, -2.866565704345703, -0.13950642943382263, -0.7675090432167053, -1.209038496017456, 0.26865479350090027, 1.810707926750183, -0.3091866374015808, -0.8052686452865601, 0.3424217402935028, -1.8606863021850586, -0.044579554349184036, 0.5608068108558655, 2.0590462684631348, -0.4359995722770691, 0.1773323267698288, 0.7899325489997864, 1.0706130266189575, -1.1029539108276367, -0.6183552742004395, -0.7740561366081238, 0.7951539158821106, 0.6651211380958557, -1.3150103092193604, -1.3083738088607788, 1.0646147727966309, 0.5807002186775208, 0.32210442423820496, 0.2562445402145386, -0.5029124021530151, 1.0760667324066162, 0.6269234418869019, 0.5587514042854309, -0.5904975533485413, -0.3550688922405243, -0.20876610279083252, 0.11690235882997513, -0.6868828535079956, 0.5530834197998047, -1.191834807395935, 1.0043691396713257, 0.3667367696762085, 0.7924008369445801, 0.7828171849250793, -0.6293715834617615, 0.06290625035762787, 2.184904098510742, -0.13563872873783112, 0.006674936506897211, 1.430484652519226, 0.04899132251739502, -0.14171092212200165, 1.1494253873825073, 0.006861364934593439, 0.16342352330684662, -0.18004395067691803, 0.16585014760494232, -0.28123345971107483, -1.2882760763168335, 1.4999877214431763, -1.5190061330795288, 0.526482880115509, 0.26444071531295776, 0.406218945980072, 0.31632548570632935, -1.1430542469024658, -0.8807968497276306, 0.3752797544002533, -0.2001575380563736, 0.3981342017650604, -0.08969132602214813, -0.15948845446109772, 1.4626778364181519, 0.5292114019393921, 0.9987912774085999, -0.06704923510551453, -0.4555591940879822, -0.23588180541992188, -1.1480363607406616, 0.5300245881080627, 1.0493985414505005, -0.4331030547618866, 0.23544572293758392, -1.520382046699524, -0.7585301995277405, 0.6166940331459045, 0.4890129864215851, 1.743228793144226, -1.606188416481018, -1.44160795211792, 0.2551283538341522, -0.018253032118082047, 0.7449324727058411, 0.6164624094963074, -0.2353212535381317, 0.42159953713417053, 0.6606313586235046, 0.5822898745536804, 0.22279231250286102, -0.19053855538368225, 1.9602012634277344, -0.04101051390171051, -0.34671247005462646, 1.6339869499206543, 0.46917250752449036, 0.7387534379959106, 0.31948670744895935, 1.007846474647522, 1.2170711755752563, 0.9288482069969177, 0.022282755002379417, -0.14939597249031067, 0.8931093811988831, 1.6226418018341064, 2.681209087371826, 0.3991522490978241, 1.1516613960266113, -0.7911732792854309, 0.018992852419614792, 0.11185318976640701, 0.5012385845184326, -0.4593910574913025, -3.0151357650756836, -0.1655188798904419, 0.08920501172542572, -0.40656447410583496, 1.5979660749435425, -1.745718240737915], [0.5455477833747864, 1.8019870519638062, 1.3589277267456055, 0.6892959475517273, -0.16233740746974945, -0.5147646069526672, -0.2908817529678345, -1.2077137231826782, 0.007378152571618557, 1.6347033977508545, 1.4585970640182495, 1.059871792793274, 0.5450977087020874, 0.49949535727500916, 1.0449912548065186, -0.4920564889907837, 0.4059702754020691, 0.03771272301673889, -0.0020047184079885483, 0.3046673834323883, 0.9562332630157471, -0.7273293733596802, -0.9167441129684448, 0.4238215386867523, 2.117968797683716, 1.2967288494110107, -1.0809457302093506, 0.9043828248977661, 0.34133583307266235, 0.31760749220848083, 0.13958211243152618, -1.6691216230392456, -0.17515410482883453, -0.03422291949391365, -2.482815742492676, 0.37892812490463257, -0.8800801038742065, -0.5926926136016846, -0.12731187045574188, 0.17362885177135468, 1.0304521322250366, 0.03729642182588577, 1.5598556995391846, 0.10317416489124298, 1.422441840171814, 0.31153640151023865, -0.09450740367174149, 0.4545891284942627, 1.2650059461593628, -0.38270318508148193, 0.2688617706298828, 0.2968285381793976, -0.19672298431396484, -1.257993459701538, 0.15468530356884003, 0.8303535580635071, 0.7204506993293762, -0.32502317428588867, -2.1938490867614746, -1.3662410974502563, -0.7973414063453674, -0.43494248390197754, -0.7702324986457825, 0.10338401794433594, 1.7590909004211426, 2.2171101570129395, 0.8717305660247803, 0.3444065451622009, 0.11855052411556244, 0.4193885028362274, 0.5019465088844299, -0.22416545450687408, -0.28222012519836426, -2.194082736968994, -0.4668862819671631, 0.4648400843143463, 0.2207728773355484, -1.0914068222045898, 0.7756266593933105, -0.5551239848136902, 0.263030469417572, 0.4540944993495941, 1.8128697872161865, 0.4893387258052826, -0.38355347514152527, -2.0145318508148193, -0.5249846577644348, 0.47656065225601196, -0.6027863025665283, 0.5224040150642395, -1.244462251663208, 1.909443974494934, -0.7809938788414001, 1.9238953590393066, -1.0798300504684448, 0.15723812580108643, -1.781518578529358, 1.1411811113357544, -0.6220040917396545, 0.20156674087047577, -0.22409392893314362, -0.3880850076675415, 1.3396598100662231, -1.0636682510375977, 1.686110496520996, -2.2709128856658936, -0.2790215313434601, -0.5795210003852844, 0.7227684855461121, -0.005922458600252867, 0.2504943013191223, 0.302030473947525, -2.110111951828003, 0.5102611780166626, -0.5381295680999756, -1.1041065454483032, 1.6163989305496216, -0.8980152010917664, 1.724457025527954, -0.757774829864502, 0.42650508880615234, 0.05239638313651085, -0.2892687916755676, -0.6219537854194641, -0.15275481343269348, -0.7943857312202454, -1.1225086450576782, 0.8245391249656677, 0.4981432855129242], [0.9106761813163757, -0.8664504885673523, 0.2549125552177429, -0.25588738918304443, -0.05411931127309799, 0.7477299571037292, 1.318062663078308, 0.8425089120864868, -0.22793620824813843, -0.4007856845855713, 0.19405168294906616, 0.7136719822883606, 0.36472925543785095, 0.6376320123672485, 0.5256218314170837, -0.12290020287036896, -2.116523027420044, 0.20044541358947754, 0.6313342452049255, 2.6275265216827393, 0.09511291980743408, 2.0725319385528564, 0.8311145305633545, -1.3115469217300415, 0.6108739972114563, 0.08441279828548431, -1.4398001432418823, -1.1420663595199585, -1.3132296800613403, 0.7952845692634583, 0.20515559613704681, 0.969173789024353, 0.16712549328804016, 0.885234534740448, 1.1206265687942505, -0.18768057227134705, -0.07383426278829575, 1.5543346405029297, -1.168469786643982, 1.2075210809707642, -1.7104918956756592, -0.0168337170034647, 0.8754652142524719, 1.3383026123046875, 0.19124165177345276, -0.9701614379882812, 1.4797699451446533, 0.12118639051914215, -0.04323543980717659, 1.0642950534820557, -0.4347226321697235, 0.8667735457420349, 0.5519024729728699, 0.8568606376647949, -0.1895100325345993, -0.2051999866962433, -0.6561627984046936, -1.9023926258087158, 0.900909423828125, 0.7342364192008972, 0.14512620866298676, -0.8491749167442322, 0.7040899395942688, 0.6597148180007935, -0.9368265867233276, -1.5746281147003174, 0.2981731593608856, -0.915563702583313, 0.059039220213890076, -0.9337360858917236, 0.40493202209472656, 0.18318966031074524, 0.17695261538028717, 0.8487979769706726, -0.018034406006336212, 0.9232119917869568, -1.2366619110107422, 0.5170744061470032, 0.6977291703224182, 1.01766836643219, 1.464985966682434, 0.11084333062171936, -1.1644405126571655, -0.1565077006816864, -1.3790745735168457, -0.7828042507171631, -0.5533466935157776, -1.379194736480713, -0.10815177857875824, -0.009521134197711945, 0.1948140263557434, -2.257598400115967, 0.6807910799980164, -1.2339518070220947, 0.9237493872642517, -0.7456833124160767, 0.5235404968261719, -0.7530587315559387, -0.7144027948379517, 0.6034702062606812, 1.3112545013427734, 1.2372578382492065, 1.82451331615448, 0.07094090431928635, -0.9186912178993225, 0.07016216218471527, -0.5763952732086182, 0.50620037317276, -0.42329126596450806, 0.843050479888916, -0.16896530985832214, 0.13992831110954285, -1.6201071739196777, -1.2357417345046997, 0.8585248589515686, -0.32168078422546387, 0.0250250156968832, -0.21647904813289642, -1.2805962562561035, -1.7707562446594238, 0.36184218525886536, 0.8098456859588623, 0.6292285919189453, 0.7783096432685852, 0.45720988512039185, 0.8893007040023804, 0.556147575378418, 0.2052096277475357, -0.4122830927371979], [-0.6566814184188843, 0.31251367926597595, 0.41116100549697876, -0.8270865678787231, 0.18569931387901306, 2.2766637802124023, 1.3620660305023193, -0.8682770729064941, 0.36731886863708496, -1.1547877788543701, -0.25285863876342773, -1.3187530040740967, 0.11307760328054428, 0.7571312189102173, 0.9673227667808533, 0.3277321457862854, 0.7288926243782043, 0.9379096627235413, 1.6803274154663086, 0.8683894872665405, 0.909993588924408, -0.03382716327905655, -1.5836364030838013, 0.7421734929084778, -0.900016188621521, -1.6982775926589966, 1.4520542621612549, 0.8196094632148743, -0.3187372088432312, 1.066948413848877, -0.7357763051986694, -0.7632739543914795, -0.28844425082206726, -0.30027273297309875, -0.14364713430404663, -0.6197079420089722, -0.8061922192573547, -0.47247037291526794, 1.1782485246658325, -0.2532753050327301, 1.5065735578536987, 0.19476726651191711, 2.7941646575927734, -0.7601100206375122, -0.7768310308456421, -0.8138473033905029, -0.49396783113479614, -0.6485686898231506, -1.452202558517456, -0.2025064080953598, 0.569943368434906, -0.3156314492225647, 0.5420132279396057, -0.3803904950618744, 1.1693907976150513, 1.4609885215759277, 0.25860413908958435, -0.5784854292869568, -0.3164847493171692, 0.28165408968925476, 0.13339652121067047, -0.3980664908885956, 0.011920677497982979, -0.7412267327308655, -0.36195287108421326, -0.07222048193216324, -0.5946032404899597, -1.5776622295379639, -1.0323612689971924, -1.8296589851379395, 0.010918847285211086, -1.8815685510635376, 0.5929232835769653, -0.33638206124305725, -1.0403510332107544, 0.28972193598747253, 0.47938525676727295, 0.6334432363510132, -0.04279317706823349, 0.6069828867912292, -1.2634477615356445, -0.2314344346523285, 0.6939756274223328, -0.23690563440322876, 0.1530005782842636, 0.8612795472145081, 0.9663437604904175, -1.7797008752822876, -0.8840751051902771, 0.17747068405151367, 0.547748863697052, 2.942537546157837, -0.05577324703335762, 1.542135238647461, 1.7393018007278442, 0.29950830340385437, 0.053249187767505646, 0.3276524245738983, -1.4901350736618042, -1.8392366170883179, 0.2868172228336334, 0.9054321646690369, -0.5331519842147827, 0.19723662734031677, -0.7802877426147461, -2.6425867080688477, -0.29186052083969116, -0.6779334545135498, -0.4559657871723175, 1.2746827602386475, -0.08731452375650406, -0.9323439598083496, 0.08094678074121475, 1.4098256826400757, 0.39838600158691406, -0.39733782410621643, -1.2167174816131592, 0.4077260196208954, -0.13092103600502014, 0.03682951256632805, -1.1084363460540771, -0.23602591454982758, -0.09526016563177109, 1.2342889308929443, -0.4837203919887543, -0.7308904528617859, 0.40775638818740845, -0.9413286447525024, -0.9680151343345642]], "dtype": "float32"}, {"__nd__": [[0.35110005736351013, -0.04308079555630684, -1.0945006608963013, 0.6602537035942078, 0.1458783745765686, -0.7287981510162354, -0.13317623734474182, -0.7529578804969788, -1.7250206470489502, -0.8355509042739868, 2.1758627891540527, 1.6577305793762207, 0.3964296877384186, 0.12842601537704468, 0.08328462392091751, 0.9830746054649353, -0.11665565520524979, 0.5078459978103638, -0.7476359605789185, -1.5056012868881226, -0.619414746761322, -0.14496518671512604, -0.5504902601242065, -0.1876215785741806, 0.5453677773475647, 0.24230512976646423, -1.8039735555648804, -0.5836993455886841, 0.17792527377605438, 0.9710934162139893, -0.24749834835529327, -0.20791690051555634, 0.955593466758728], [0.7374642491340637, -1.081527829170227, 0.36991551518440247, 0.519810140132904, -0.02242845855653286, -1.7035613059997559, 0.7737200856208801, 0.0220487117767334, -0.31318703293800354, 0.21066078543663025, 1.2353850603103638, 0.5251399278640747, 0.8619256019592285, -0.8023272752761841, -0.6139389872550964, -0.5100371241569519, -0.5734703540802002, 0.4302349090576172, 0.2549557685852051, -0.7206171751022339, -1.9748284816741943, 0.48705586791038513, -0.2426096498966217, -0.11281251162290573, -2.2364182472229004, 0.15248316526412964, 0.9018010497093201, 1.0288801193237305, 0.4714552164077759, -0.14573001861572266, -1.7269757986068726, 0.5164172053337097, 0.566505491733551], [-1.419476866722107, 0.04698578640818596, -0.1956295222043991, -0.4026632010936737, 0.5366738438606262, 1.0462275743484497, 1.9768363237380981, 1.1707056760787964, 0.16891756653785706, -0.7796053290367126, 1.4118764400482178, 0.11416082829236984, 2.0909106731414795, 0.12271519750356674, -1.018465280532837, -1.3913925886154175, -0.7097591757774353, -1.8010302782058716, -0.291639119386673, -0.54155433177948, 1.1766506433486938, -1.1698877811431885, 0.022822314873337746, -0.7017495632171631, -0.7451348304748535, 1.226799726486206, 0.5072534680366516, -0.596077561378479, -0.4404924809932709, 1.6815757751464844, 1.0652035474777222, -0.024228937923908234, -0.06181610748171806], [-1.094191551208496, -0.5217667818069458, 0.1248137578368187, -1.320427417755127, -0.17232286930084229, -0.47711798548698425, -0.10865117609500885, -1.7007217407226562, -1.1934906244277954, 0.8619639277458191, 0.548487663269043, 1.1384042501449585, 0.6664813756942749, 0.4501771032810211, 0.6069912314414978, -0.346528023481369, -1.3153948783874512, 0.0018155159195885062, -1.5025888681411743, -0.354806512594223, 1.7323635816574097, 1.5590280294418335, 0.0021477164700627327, 0.056702010333538055, 0.2044236958026886, -2.0690741539001465, 0.24442659318447113, 0.812335729598999, -0.0418526865541935, -0.6051401495933533, 0.5880097150802612, 0.19919933378696442, 1.2576969861984253], [1.0614607334136963, -0.5239485502243042, 2.200709581375122, 1.0728939771652222, 1.4354405403137207, 0.05425934121012688, 0.21363253891468048, -1.2078267335891724, 0.7257620692253113, 0.369912713766098, 1.5801935195922852, -0.13056713342666626, -0.2637191116809845, -1.29366135597229, -0.7015987634658813, 0.38883182406425476, -1.5193006992340088, 0.18356606364250183, 0.6483988165855408, -0.9133118391036987, -1.2560559511184692, 0.15390744805335999, -1.0563819408416748, -0.9113949537277222, 0.2129695564508438, 0.10384736955165863, 0.6047238111495972, 1.3408805131912231, -0.3006925880908966, -1.0726786851882935, 0.18664421141147614, 0.4926016926765442, -2.6200058460235596], [-1.3634426593780518, 2.0654959678649902, 0.042137909680604935, 0.5566756725311279, 1.051346778869629, 1.3993515968322754, -0.604536235332489, -0.41373080015182495, 0.7910750508308411, -0.09726615995168686, -1.749402642250061, -0.8282641768455505, 0.7715235352516174, 0.01731201633810997, 0.3035357594490051, -0.015074250288307667, 0.22693273425102234, -0.20282825827598572, 0.14910390973091125, -1.0468782186508179, 1.0804697275161743, 0.42484328150749207, -1.3133773803710938, -1.5186868906021118, -0.20604638755321503, 0.6160690784454346, -2.0905356407165527, -1.080831527709961, -0.7359897494316101, 0.2895714342594147, 1.2035260200500488, 0.25193679332733154, 0.5089918971061707], [0.5977388024330139, 0.7862098217010498, 0.3430456221103668, -0.08669959753751755, -0.9604044556617737, -1.3984383344650269, -0.6617690324783325, -0.14045079052448273, -1.4975082874298096, -1.2360198497772217, -0.22138534486293793, -0.28747573494911194, 0.23429608345031738, 0.8211619257926941, 0.6659840941429138, -1.0222548246383667, 0.48780807852745056, 1.0724997520446777, -0.1323869377374649, 0.2898419201374054, 2.588158130645752, 0.8845250010490417, 1.4023751020431519, 0.9849179983139038, 0.45673343539237976, 1.3057987689971924, -1.6294456720352173, -0.8436009883880615, -1.002180576324463, -0.277072012424469, -1.5196110010147095, -0.6129193305969238, -0.350469172000885], [-0.9041946530342102, 1.312047004699707, 0.8756086826324463, -0.7448496222496033, -0.737736165523529, 0.3991720974445343, 1.1135311126708984, 0.623977541923523, 1.0663580894470215, 1.3280407190322876, 0.14892393350601196, 1.0115606784820557, 0.6475656032562256, -0.9441459774971008, -0.32557299733161926, -0.21079085767269135, 0.9183661937713623, 1.0998804569244385, -0.37386855483055115, -0.05382738262414932, 0.8796180486679077, 1.2519989013671875, -1.5590088367462158, -0.508594810962677, 1.2318588495254517, 0.8763511776924133, -1.2601407766342163, 0.8744486570358276, -0.3105306029319763, -1.1027518510818481, 0.8500670194625854, 0.9651615023612976, 1.0642036199569702], [-0.7927705645561218, 0.44779086112976074, 0.4659573435783386, 0.15364432334899902, 0.6744970083236694, 0.3237125277519226, -0.821290910243988, -1.308868169784546, -0.35245972871780396, 1.9210293292999268, 0.7838588953018188, 1.0942069292068481, -0.5344336032867432, -0.02572798915207386, -1.9606326818466187, 0.45063456892967224, -0.1756548434495926, -1.3086798191070557, -0.3045724332332611, -0.8800277709960938, -0.5739072561264038, -0.0477704256772995, 1.3401975631713867, 1.5619430541992188, -0.5327161550521851, 1.5656031370162964, -0.3157613277435303, -2.0287556648254395, 2.346573829650879, 1.384908676147461, -1.8338192701339722, 1.4374377727508545, -0.019580025225877762], [-1.0906709432601929, -0.14306914806365967, -0.5323091149330139, 0.6970222592353821, -0.8442310094833374, -1.6113470792770386, 0.4018371105194092, -1.0708582401275635, 0.009720875881612301, -0.9860356450080872, -1.6615009307861328, -0.4599980413913727, -2.269860029220581, 0.3160535395145416, -0.1166171282529831, 1.3584898710250854, -0.3173980116844177, 0.4532391130924225, 1.8365368843078613, 1.3330681324005127, -0.4912716746330261, 0.04187272489070892, -0.8684136271476746, -0.3827258348464966, 0.6154513955116272, 2.8541507720947266, -2.2043797969818115, -0.48866939544677734, -0.7369415760040283, 0.6198988556861877, 1.07840096950531, 1.4904823303222656, -0.6853896379470825], [-0.15763044357299805, -1.4374388456344604, 2.588291645050049, 2.4523732662200928, 2.4305472373962402, 0.8837094902992249, 3.3071117401123047, 0.2817040681838989, -0.873677134513855, 1.072043538093567, -0.17915397882461548, 1.4920742511749268, -0.3600997030735016, -0.13612785935401917, 0.9275927543640137, 0.453164666891098, -0.01601308211684227, -0.11059223860502243, -2.5881857872009277, -0.2559192180633545, -1.3178763389587402, -1.4376810789108276, 1.5357229709625244, -1.594525694847107, -0.12474199384450912, -1.466467022895813, -0.39862218499183655, -0.5891443490982056, 0.9419853091239929, -1.584644079208374, 0.6782703399658203, 2.649472236633301, 0.8713676333427429], [-0.3022308647632599, 0.046363845467567444, 1.590187668800354, 0.16407430171966553, 0.295533686876297, -0.47928866744041443, -0.2190978229045868, -0.29144513607025146, -1.1416136026382446, -1.6943591833114624, 0.8218206167221069, 0.1340979039669037, -2.2417705059051514, -0.9810613989830017, -0.8659225702285767, -0.5512055158615112, 0.29952362179756165, 1.064171314239502, 0.5818261504173279, 1.2980071306228638, 1.00702702999115, 0.8138988018035889, -1.904603123664856, 1.2301254272460938, 1.040201187133789, 0.21785858273506165, 0.009549496695399284, 1.5410722494125366, -0.06495263427495956, 2.2313172817230225, 0.41901829838752747, -0.5798557996749878, 0.5438204407691956], [0.983697235584259, -1.7805663347244263, -0.6915571093559265, 0.3820837438106537, -1.9454463720321655, 0.19480395317077637, 0.42326056957244873, -0.511076033115387, -0.603905975818634, -0.3830021321773529, 0.2728869616985321, -0.5089507102966309, 0.5622379183769226, -0.055351607501506805, -0.6630816459655762, -0.07524964213371277, 1.1920185089111328, 0.5286182761192322, -0.3904673755168915, 0.4435385763645172, 0.40318942070007324, -1.8510513305664062, 1.0975675582885742, -1.045815348625183, -0.3795240521430969, 1.0774879455566406, 1.9181770086288452, -0.5363914966583252, -1.1836087703704834, 0.6956856846809387, 2.034860134124756, 0.2929344177246094, 0.2275829017162323], [1.1564464569091797, 0.6602643132209778, -1.0633372068405151, 0.7303277254104614, -1.6772828102111816, 0.028265463188290596, -1.1403695344924927, 0.6521100401878357, 0.24959386885166168, 1.0717027187347412, -0.16058629751205444, 0.4497740566730499, -0.26278841495513916, 0.5343912243843079, -0.521264910697937, -0.3949875235557556, -0.015271625481545925, 0.8692829608917236, 1.082214117050171, 0.6336584091186523, -0.25585561990737915, -0.03005087561905384, 0.7077952027320862, 1.2164427042007446, 0.2598475515842438, 0.8706726431846619, -0.8156763911247253, 0.8901683688163757, -0.2443372905254364, -0.4019263684749603, -0.22575637698173523, -0.3230631947517395, 2.0113463401794434], [-0.4524557590484619, -0.013033172115683556, -0.5763679146766663, 0.42798373103141785, -0.9954761862754822, -0.9276735782623291, 0.07913326472043991, 0.3303242325782776, -0.7926754951477051, -1.6266027688980103, -0.7529773116111755, 2.2869021892547607, 1.0293384790420532, -0.3067677319049835, 0.3732515573501587, -0.6639970541000366, -0.37641698122024536, 1.5400723218917847, 0.44715598225593567, -0.07048826664686203, 0.6851649880409241, -1.5615031719207764, -0.02757384069263935, -1.2831488847732544, 2.054265260696411, 1.7256829738616943, -0.05540290102362633, -0.9191561341285706, 0.3040686547756195, 0.8510653376579285, 0.7873141765594482, 0.1724901646375656, 1.7995127439498901], [0.12781719863414764, -0.603524923324585, -0.35287225246429443, 0.4927023649215698, 0.3309203088283539, -0.9472194314002991, 0.18375997245311737, 1.201657772064209, 0.014408300630748272, -0.6415264010429382, 1.2616230249404907, -0.30766400694847107, 1.9085912704467773, -1.317627191543579, 0.3432866036891937, -0.25762927532196045, -0.31196328997612, -0.26673007011413574, 0.22054210305213928, 0.6627910137176514, -0.6514625549316406, 0.9089372754096985, 1.0655765533447266, 0.34826838970184326, -0.24988383054733276, -0.8251538872718811, 0.05844350904226303, 0.35130780935287476, 1.2770460844039917, 0.22991101443767548, 0.5347352623939514, 0.07981949299573898, 0.6157707571983337], [-1.6568491458892822, -1.546143651008606, 0.4486997425556183, -1.3828521966934204, 0.6674329042434692, -1.099123477935791, 0.29435601830482483, 1.579480767250061, 0.3820136785507202, -0.546758770942688, 0.0473005510866642, -0.17505280673503876, -0.6363311409950256, -1.9002090692520142, -0.6028733849525452, -1.5106221437454224, 1.0320994853973389, -1.8946701288223267, -0.2973212003707886, 0.5319912433624268, 0.19814243912696838, -0.5686327219009399, -1.005205512046814, -1.053719401359558, 1.0185836553573608, -0.3645783066749573, 1.2065240144729614, -0.708046555519104, -0.05872572958469391, 0.21054822206497192, -0.8603798151016235, 0.11828803271055222, 0.6105568408966064], [-1.072798252105713, 1.2573899030685425, 0.8489379286766052, 0.05018230527639389, -0.9791427850723267, -1.0697745084762573, 0.5757265686988831, 0.8252840042114258, -0.23444466292858124, 0.52976393699646, 1.4907928705215454, 1.0953477621078491, -1.149105191230774, 1.7198584079742432, 0.2732926607131958, 0.4352915287017822, -0.9514407515525818, -1.8396576642990112, -0.7838221788406372, 0.651671826839447, -1.1849457025527954, 0.03251756355166435, 1.1532233953475952, 2.1852364540100098, -0.17278680205345154, 0.0006318060914054513, -0.5848444700241089, 1.3637266159057617, -0.3313806653022766, -0.20745761692523956, 0.04429541528224945, -0.6807301044464111, -0.0496126227080822], [0.6994759440422058, 0.514082670211792, -0.05506142973899841, 0.20249463617801666, 0.24502314627170563, -1.3493599891662598, 0.18650776147842407, -0.23291781544685364, -0.06507117301225662, -0.18281885981559753, 0.9889898300170898, -1.3396888971328735, -1.0537980794906616, 1.456531047821045, -2.5343902111053467, 0.37807324528694153, 0.4025014042854309, 0.6397192478179932, -1.382893681526184, -1.088173747062683, -0.793295681476593, -1.1355266571044922, -0.4213021397590637, 0.5102750658988953, 2.304450750350952, -2.036262035369873, -1.0832186937332153, 1.4251909255981445, 0.849269449710846, 0.6847366690635681, 1.5192532539367676, 0.2612205147743225, -1.5568429231643677], [0.23991888761520386, -1.3361735343933105, -0.3286231458187103, 0.7318869233131409, 0.3453039824962616, -0.49693068861961365, -0.5287353992462158, -1.0542869567871094, 0.15998610854148865, -0.9416332840919495, -0.3731478452682495, -1.0553704500198364, 1.3191945552825928, 1.3212929964065552, -0.8139743804931641, 1.3516438007354736, 0.6842178702354431, 1.0307879447937012, 0.23260430991649628, -0.18970435857772827, 0.8307548761367798, 0.18247631192207336, 0.5681174993515015, 0.05505348742008209, 0.37929514050483704, 0.5208815932273865, -0.14356565475463867, 0.23433950543403625, 0.5457524657249451, 0.10088546574115753, 2.049497127532959, 0.40051689743995667, -0.3737148344516754], [-1.0069642066955566, -1.5470490455627441, 0.8175817131996155, -1.4129278659820557, 1.3597365617752075, 0.8240259885787964, 0.4013655185699463, 1.7803593873977661, -0.8251233696937561, 0.39532315731048584, 0.9252488613128662, 0.39986997842788696, 0.14084921777248383, 0.6071171760559082, -0.251102477312088, -0.08494023233652115, -0.37391120195388794, 0.4647285044193268, -0.7166293263435364, 0.19853182137012482, -0.10442102700471878, 1.0464340448379517, 0.48489177227020264, 0.4827665090560913, 1.5711160898208618, 1.9412654638290405, -1.2784113883972168, 0.0021094647236168385, 0.6930782794952393, 0.7672690153121948, 0.2426328957080841, 0.5014493465423584, -1.7423193454742432], [-2.35286283493042, -0.09590117633342743, 1.4479563236236572, -0.6999953985214233, 1.3507587909698486, 0.13278943300247192, -0.8727145791053772, 1.5950947999954224, 0.3868137001991272, 0.10390355437994003, -0.335248202085495, 1.121300220489502, -0.26071399450302124, 0.6104711294174194, -1.4779528379440308, 0.1356843262910843, -0.860539436340332, -0.563424289226532, -1.3315502405166626, -0.22613324224948883, -1.5763345956802368, -0.4219655692577362, -1.4370168447494507, 1.4687888622283936, -1.3358489274978638, 0.075694739818573, -1.1625394821166992, 0.42035242915153503, 1.757583737373352, 0.050020672380924225, 0.13283564150333405, -1.222242832183838, 0.31937023997306824], [-0.04253416880965233, 1.2382891178131104, 0.25476717948913574, -0.6732342839241028, -0.6751304864883423, 0.8012794256210327, 1.5378170013427734, 2.4455161094665527, -1.3009542226791382, 0.935554563999176, -0.6812136173248291, 0.20564642548561096, 2.1329617500305176, 1.9646278619766235, -0.7180155515670776, -0.3051614761352539, -0.6521025896072388, 1.7008365392684937, 1.3521970510482788, -1.413870096206665, -0.32901620864868164, -1.271705985069275, 1.5901366472244263, 2.1270220279693604, 0.8074339032173157, -1.6967065334320068, -0.8515740633010864, 2.1035094261169434, 0.6288080215454102, -0.1712881177663803, -0.14815109968185425, -1.760689377784729, 0.6410101652145386], [0.3972914516925812, 0.9700314402580261, 0.507548451423645, 0.7745484709739685, 0.8822687864303589, 0.6919260621070862, 0.09685634076595306, 0.372188001871109, -2.0530436038970947, 1.1747925281524658, 0.9901037812232971, 0.3534489870071411, 0.9064354300498962, -1.8604331016540527, -2.495640993118286, -0.6442386507987976, -0.04268854856491089, -1.0979596376419067, -1.5649113655090332, -0.4490262567996979, 0.4829982817173004, 0.39645636081695557, -1.586576223373413, 0.2686482071876526, -0.8649810552597046, 0.6046118140220642, 1.6067142486572266, -0.9780344367027283, 1.0577739477157593, -0.41383254528045654, -0.48629629611968994, 0.3606758415699005, 0.39748913049697876], [-0.5199348330497742, -0.8425086140632629, 0.78789883852005, 0.5061346292495728, 0.610130250453949, -0.4617736339569092, -0.6748324036598206, 1.1137175559997559, 0.44922223687171936, -0.9093421697616577, -1.079314947128296, -1.2479755878448486, -0.4960213005542755, -0.6786670088768005, 2.3847131729125977, 1.7676031589508057, 0.48409637808799744, 0.8200053572654724, 0.3454605042934418, 1.940064549446106, -1.8190395832061768, -0.259371817111969, -1.3638699054718018, 0.3142569363117218, -0.03754252940416336, 0.04136195778846741, 0.2860746383666992, -0.28302982449531555, -0.6921939849853516, -0.31862643361091614, -0.4401623606681824, -0.6295139789581299, 0.603141188621521], [0.5662294626235962, -0.558317244052887, -0.765507161617279, -3.2714223861694336, 0.14001552760601044, 0.2864672839641571, 1.19066321849823, -1.1886659860610962, 1.439176082611084, -1.6267796754837036, -3.31889271736145, 1.2248518466949463, -1.3287549018859863, 0.31437286734580994, -0.8369719982147217, -1.4833427667617798, -0.04252023622393608, 0.8293746709823608, 1.2534645795822144, -0.787590503692627, 0.3984854817390442, 1.0710817575454712, 0.131024569272995, -0.7821962833404541, -0.04354517534375191, -0.6475561857223511, -0.4091111123561859, -1.8373156785964966, -0.5945524573326111, 1.0084999799728394, 0.7271589040756226, 1.2116446495056152, -0.5275571942329407], [-0.5484014749526978, -0.02969280630350113, -0.8987472057342529, 0.0009633942972868681, 1.5631496906280518, -0.13732042908668518, 0.5180932879447937, 0.7764011025428772, 0.4688623249530792, 0.15035805106163025, -1.9435592889785767, 0.41912099719047546, 1.2347253561019897, 0.9949960112571716, -0.9153723120689392, 0.8926491141319275, -0.6501320004463196, 0.3083958923816681, -0.054313674569129944, 0.2473558783531189, 1.472051978111267, 0.23739512264728546, -0.16050855815410614, -0.89085453748703, -1.617616057395935, -0.7441375851631165, 0.9576072692871094, 0.35536840558052063, -1.1332582235336304, -0.6217740178108215, 1.2128633260726929, 1.3766857385635376, 0.21735453605651855], [1.7582155466079712, 0.9459636807441711, -0.28927305340766907, -2.3619723320007324, -0.9975079298019409, 0.5483460426330566, -1.2368338108062744, -0.7340773940086365, -0.09851787984371185, -0.21366466581821442, 0.9277781844139099, -0.3628004789352417, 1.2840062379837036, 0.8129667639732361, -1.5695124864578247, -0.5000457763671875, -0.7788894176483154, 0.23937900364398956, 1.548251986503601, 0.5643690228462219, 2.0305025577545166, -1.5521637201309204, -0.1721900850534439, 0.433919757604599, 0.43044430017471313, -1.8217841386795044, -0.3843596577644348, 0.22505587339401245, -1.2142871618270874, 1.5478010177612305, 0.7543492913246155, 0.32346460223197937, -0.24144500494003296], [0.8871951103210449, -1.9749996662139893, -0.5781194567680359, 0.38089653849601746, -0.9140597581863403, -0.6586810350418091, 0.5200831294059753, 1.2320812940597534, 1.7039035558700562, -0.5335063338279724, 0.9851098656654358, 1.7385900020599365, 0.2624518871307373, 0.8455538749694824, -0.17520977556705475, 0.434020459651947, 0.4757922887802124, -0.3622228503227234, -0.7578611373901367, -1.61729097366333, -0.3600519299507141, -0.08709156513214111, 0.4547169506549835, -0.05009710416197777, -1.2596126794815063, 0.5430675148963928, 0.7850272059440613, 1.4094356298446655, -0.7245320081710815, 0.5775344967842102, -0.34047964215278625, 0.6931240558624268, -0.08573317527770996], [-1.409731388092041, -1.4299302101135254, -0.4118475914001465, -1.2464524507522583, 0.2861160635948181, -1.191727638244629, 0.7785981297492981, -0.2900903820991516, -0.9137134552001953, 0.5411574840545654, 0.05521894618868828, -0.2365289032459259, 0.08624055236577988, 0.032869137823581696, 0.8867127895355225, -0.3460547924041748, 2.203155040740967, -0.15005777776241302, 0.030805395916104317, -1.454083800315857, -0.7053066492080688, -1.4787367582321167, -0.24131666123867035, -0.02375577576458454, 0.6003617644309998, 0.3313310742378235, 0.7346006035804749, -0.7088345289230347, 0.06153767928481102, 0.6779071092605591, 2.4344351291656494, 0.3732188642024994, -0.6858001947402954], [-1.2240053415298462, -0.18573987483978271, -1.563415288925171, 0.37133848667144775, 0.908231794834137, 0.06258796155452728, -1.431976079940796, 0.1209290474653244, 1.0527881383895874, 0.2674913704395294, -0.6278205513954163, -1.2724649906158447, 0.4261743426322937, -0.3231886625289917, -0.2179868221282959, 0.8853915333747864, 0.583225667476654, -1.7105225324630737, -0.19627676904201508, 0.8311521410942078, 0.9971933960914612, -1.3936865329742432, 0.19539496302604675, -0.2990826964378357, -1.3702722787857056, -0.31157949566841125, -1.427528738975525, 0.74703049659729, -0.09259166568517685, 0.22553619742393494, 0.6745755076408386, -0.7960572838783264, 0.8967204689979553], [-0.8163827061653137, -2.4213531017303467, 0.5522070527076721, 0.6540455222129822, 0.7696876525878906, 0.07451074570417404, 0.3837851881980896, -1.183268666267395, 1.436429500579834, 0.6242430806159973, -0.9635675549507141, 1.5482580661773682, 0.8843828439712524, 0.11220858246088028, -1.2571301460266113, -1.6491787433624268, 0.3488497734069824, 0.4863910377025604, 0.6279327273368835, -0.9015874862670898, 1.621993899345398, -0.30750077962875366, -1.4314196109771729, 0.9963701963424683, 1.582057237625122, -0.3921184539794922, 0.03422805294394493, -1.8805180788040161, -1.9269787073135376, -0.514121949672699, -1.1610963344573975, -0.2539788484573364, -0.002735758200287819], [-0.7907220721244812, -0.14975188672542572, 0.5857394933700562, 1.564225196838379, -0.6513381004333496, -0.27600404620170593, 0.4697645306587219, -1.0971496105194092, -0.593155562877655, -0.6279361844062805, -1.3872625827789307, 0.9514235854148865, -0.3876895308494568, 1.6614835262298584, -0.6383916139602661, 0.29524117708206177, 0.40658965706825256, 0.8642650246620178, -0.798389732837677, -0.9209113717079163, 1.4872612953186035, 0.3551647663116455, -0.2784391939640045, 1.0024752616882324, 1.6168991327285767, -0.7010482549667358, -0.4030751585960388, -0.5833378434181213, 1.3360795974731445, 0.09043353796005249, -1.5055955648422241, 0.047625668346881866, -1.7420927286148071], [0.9214717149734497, -0.6831167340278625, -0.07541710138320923, -0.07877495139837265, 1.632537603378296, 1.2947089672088623, -2.178718328475952, -0.31534504890441895, 0.9704815149307251, -0.27546924352645874, -1.6301273107528687, 1.4254393577575684, 0.8580293655395508, -0.5855276584625244, 0.7386855483055115, 0.12108100950717926, 0.2955492436885834, -0.2744699716567993, 0.582273542881012, -0.5145281553268433, -1.8647090196609497, -0.6308729648590088, 0.24330079555511475, 0.7246754169464111, -0.37415945529937744, 0.1951705664396286, 0.5707943439483643, -0.00028000507154501975, -1.3735957145690918, -0.45229029655456543, 0.11749289184808731, 0.4634200632572174, 0.03701071813702583], [0.8724401593208313, 1.9627525806427002, 0.07056564837694168, 0.6285184621810913, -0.5239344239234924, -0.4745926260948181, 1.337911605834961, 0.16487637162208557, -2.0210957527160645, 0.3544553518295288, -0.9940249919891357, 1.098057508468628, -0.6380389332771301, -1.275495171546936, -1.0637911558151245, 0.2209477573633194, -0.25578153133392334, 1.0155454874038696, 0.8518917560577393, 0.4309910237789154, -1.1940327882766724, 1.6714469194412231, -1.3685494661331177, 0.4440020024776459, -0.9881219863891602, -0.009898745454847813, -0.1886003464460373, 0.4963837265968323, 0.19258388876914978, 1.297800064086914, -0.9905046224594116, -0.2827170789241791, -0.024509193375706673], [0.29587510228157043, 0.6107804179191589, -1.4655576944351196, 0.2218846082687378, -0.26042091846466064, -2.1900007724761963, 0.8232541084289551, -0.8813812136650085, 0.6904255151748657, -1.1768797636032104, 0.8163437247276306, -1.479925513267517, 0.9634552001953125, -0.18797016143798828, 0.026370372623205185, -1.2002369165420532, 0.6951911449432373, -0.19662344455718994, 0.8851033449172974, 0.5079058408737183, -0.681583046913147, -0.09005650132894516, -0.07023252546787262, 0.21435832977294922, -1.5272010564804077, 1.1689180135726929, -0.8413190841674805, -0.6757771968841553, -0.07589859515428543, -0.0744173675775528, -0.5988458395004272, 0.8053846955299377, 0.08401903510093689], [-0.07205241918563843, -0.5940456390380859, 0.7499802708625793, 2.0567641258239746, 0.10881593078374863, 0.0026347567327320576, 1.5977342128753662, -0.3817472457885742, -0.20886175334453583, 1.0547544956207275, 1.715545415878296, 0.4013507664203644, 0.9738634824752808, 1.7379437685012817, 0.37968987226486206, -1.201253056526184, -1.182491660118103, -0.5216261148452759, 0.9356104135513306, -1.374393343925476, -0.2667698860168457, 0.3359583616256714, -1.8557838201522827, -1.2330623865127563, 0.7437037825584412, 0.34173738956451416, 0.2732812762260437, -0.8139917850494385, -0.14463083446025848, -0.021348420530557632, 1.181406021118164, 0.25460413098335266, -1.7271002531051636], [0.030823124572634697, 3.0667500495910645, -0.5482151508331299, -0.12550075352191925, -0.517178475856781, 0.6542173624038696, -0.16065408289432526, -0.7060367465019226, -1.680627465248108, 1.2335708141326904, 1.321178674697876, 0.09207528084516525, 0.5230098366737366, 1.0198081731796265, -0.004421653691679239, 0.16627739369869232, -1.116631269454956, -1.1740645170211792, -0.32015904784202576, -1.1776014566421509, -0.2959151566028595, -0.3984885811805725, 0.7811284065246582, 0.26696041226387024, 1.2004879713058472, 0.453408420085907, -0.6008216142654419, -0.376956045627594, 3.218442440032959, 0.5198009610176086, -0.4015156328678131, -0.5309386849403381, 0.21152716875076294], [1.6136114597320557, 0.5064384341239929, 0.4396227300167084, 0.7993765473365784, -0.017735356464982033, 0.2971370220184326, -0.5149522423744202, -0.2725052535533905, 1.1671353578567505, -0.0010403464548289776, 0.7100984454154968, -2.121652364730835, -1.9163497686386108, -2.3925342559814453, -1.300330638885498, -0.48606565594673157, -0.022118018940091133, 0.939387321472168, 0.6810906529426575, -1.5532315969467163, -1.388086199760437, -1.7034884691238403, 1.403088927268982, -1.7250276803970337, -2.80216121673584, 1.1015069484710693, -0.6868181824684143, -0.4621959626674652, -0.21100175380706787, -0.8398500680923462, -1.6842914819717407, -0.4332016408443451, -1.4796830415725708], [0.5641854405403137, 0.16827034950256348, 1.2477434873580933, 0.12145259231328964, 1.3804460763931274, -0.9698811173439026, -0.24506282806396484, 0.13575705885887146, 0.41262179613113403, 0.3676654100418091, -1.2183141708374023, -1.6600450277328491, -0.6464192271232605, 0.17474153637886047, -2.179196357727051, 1.1925603151321411, 0.24282291531562805, -0.8291485905647278, -1.2538050413131714, -1.9255235195159912, -1.0491931438446045, -0.3019251823425293, 0.8139113783836365, 0.4745716452598572, 0.8899086117744446, -0.4909144341945648, 1.3266091346740723, 1.0162771940231323, 0.649213433265686, 1.3920872211456299, -1.1010677814483643, 0.7360895276069641, 0.16614000499248505], [0.40806013345718384, 1.6388529539108276, 0.19858892261981964, 0.906408429145813, -0.9630639553070068, -0.020919179543852806, 0.21936564147472382, 1.720069408416748, 0.20905080437660217, -0.2093522846698761, 1.7221543788909912, 1.0798649787902832, -0.5792176127433777, -0.0026636766269803047, -0.2052880972623825, 1.1420626640319824, -0.01930481567978859, 0.5933314561843872, -0.32196107506752014, -0.6150482892990112, 0.9212111234664917, 0.1761147528886795, 0.5950316190719604, 0.9566172957420349, 1.9487172365188599, 0.7547034025192261, 1.2337263822555542, 0.16449163854122162, -0.029666028916835785, -0.35794928669929504, -1.6569583415985107, -0.42019960284233093, 0.5737804174423218], [1.4636318683624268, 0.561917781829834, 1.3863794803619385, -1.08921217918396, 0.5842879414558411, 0.38358306884765625, -0.6158950924873352, 0.2584059238433838, 1.67965567111969, 0.7825778126716614, -0.3237902522087097, -0.7485547065734863, 0.038811590522527695, 1.4579365253448486, 0.21693599224090576, 0.05994129180908203, -1.3722401857376099, -0.9443617463111877, 0.012228576466441154, 2.661778450012207, -0.7425083518028259, 1.7449560165405273, 0.5235162973403931, 0.4112113118171692, 0.3749140501022339, 0.4009902775287628, 0.23354466259479523, 0.4965912401676178, -0.2529325783252716, 0.3918115198612213, -0.28277134895324707, 0.5629238486289978, 1.3631125688552856], [-1.6552269458770752, 1.1287972927093506, -0.27185606956481934, -1.086808443069458, -0.36599084734916687, -0.36298298835754395, 0.2018546760082245, 0.7606580853462219, 0.9413743615150452, 1.308838129043579, -0.41826069355010986, 1.4538873434066772, -0.9270175695419312, 0.43396955728530884, -0.44949302077293396, -1.8622833490371704, 0.47237080335617065, 2.0125575065612793, 0.6891674995422363, -0.786088764667511, -0.5618945360183716, 0.5735965967178345, -0.3661094307899475, 0.5082798600196838, -0.47173863649368286, -0.7807385921478271, -0.2691831588745117, 0.6904184818267822, 0.029218366369605064, -1.0041990280151367, 0.11915434151887894, 1.3691942691802979, -0.08893294632434845], [-0.3209165632724762, 1.4310518503189087, 1.5732238292694092, 1.7671936750411987, 0.1606428474187851, -1.4705355167388916, -0.02375248447060585, 0.3437815010547638, 1.712934136390686, -0.7293741703033447, -2.7736024856567383, 0.9262574315071106, 0.4644162952899933, 1.5100520849227905, 0.47948357462882996, -0.8275204300880432, 2.6707189083099365, 1.0549097061157227, -0.6622111201286316, -1.1901829242706299, 0.8171775341033936, 0.9637707471847534, -0.5182875394821167, 0.892937183380127, -0.15435200929641724, 1.5590648651123047, -0.5962697267532349, 1.5384379625320435, -0.3979184329509735, 0.8406184315681458, -1.5314711332321167, 0.38130152225494385, -0.867234468460083], [-1.3197702169418335, -1.0604665279388428, -0.05514640733599663, -0.06078948453068733, 0.8643617033958435, 0.13834232091903687, -0.49862420558929443, 1.2356384992599487, -0.7720643281936646, -1.1347582340240479, -0.90105801820755, -0.28896012902259827, 0.26201680302619934, 2.062981128692627, 0.014704410918056965, 0.20809978246688843, 0.23102052509784698, -1.5308390855789185, 0.759590208530426, -1.8169608116149902, 0.3212873637676239, -0.23883402347564697, -0.2372884601354599, -1.747959017753601, 1.2105472087860107, 2.2310495376586914, -0.7460408210754395, -0.011569725349545479, -0.7213853001594543, 0.22466939687728882, -0.8237732648849487, 0.49321672320365906, 0.06990598887205124], [-1.0013242959976196, 1.0946686267852783, -1.1042371988296509, -0.5463682413101196, -1.281480312347412, -1.414160132408142, 0.8783903121948242, -0.13060209155082703, -0.1636420041322708, 1.6914691925048828, 0.00825156643986702, 1.1766687631607056, 0.1559031456708908, 0.047985758632421494, 0.5054572820663452, 1.5183485746383667, 0.4379256069660187, 1.008378267288208, -0.3006948232650757, 0.38160061836242676, 0.4326808750629425, 0.994109570980072, -0.7862859964370728, -2.286621570587158, 0.3147158920764923, 0.8684301972389221, 0.109474778175354, 0.12595154345035553, -1.135632038116455, 0.6145911812782288, -1.1657785177230835, -1.5617183446884155, -0.9269543290138245], [-0.4426741302013397, 1.5803745985031128, -0.39542126655578613, -0.5381747484207153, 0.3719242513179779, 0.18001601099967957, -1.434756875038147, 0.9338117241859436, 1.543035864830017, 0.22372466325759888, -1.1114633083343506, -0.6448453068733215, -0.41199252009391785, 0.40602394938468933, 1.4990522861480713, -0.15664862096309662, 0.42168140411376953, -0.49847379326820374, 1.0276936292648315, 0.20841015875339508, 0.09268689900636673, 0.6414226293563843, -1.7954350709915161, 0.19216227531433105, 0.43818941712379456, -1.7386908531188965, 1.1607130765914917, 0.4357704818248749, 0.5562139749526978, 0.392121285200119, -0.6761250495910645, 1.083249807357788, -0.2515069842338562], [-1.2341731786727905, 0.31339722871780396, -1.3374173641204834, 0.2987551689147949, -0.6806028485298157, 0.093644879758358, -0.2511463463306427, -1.2788923978805542, -0.25329312682151794, -1.3607783317565918, 1.041235327720642, -2.0491909980773926, -0.6471297144889832, 1.5084925889968872, 0.04450913891196251, -0.8285750150680542, -1.1598975658416748, -0.2642919719219208, 0.2921864986419678, 1.1957918405532837, 2.2293529510498047, 0.5613043904304504, 1.0211219787597656, -0.36348438262939453, -0.4812619090080261, -0.23012734949588776, -2.185997724533081, -0.3408379852771759, 0.09944593906402588, -0.5557748675346375, 1.6003810167312622, 0.21191026270389557, 0.10801951587200165], [0.4430082142353058, -0.8099749088287354, -0.11001603305339813, 0.22775444388389587, -0.9862217903137207, 0.5219230055809021, -1.426632285118103, -0.43104633688926697, 0.013879011385142803, 0.7261750102043152, -0.16283299028873444, 0.011525875888764858, 2.589988946914673, -1.3074432611465454, 0.3227439522743225, -1.1724748611450195, -1.9022964239120483, -1.2092359066009521, -0.7108798027038574, 0.40382295846939087, -1.213451862335205, -0.6172146797180176, -0.15953394770622253, 2.080366849899292, -0.2675321698188782, -1.076151728630066, -0.04700460284948349, 1.46381676197052, 1.7670176029205322, 1.0323628187179565, -1.414547324180603, 1.4672164916992188, 0.22585907578468323], [1.7398275136947632, -0.011451354250311852, -2.553976058959961, 0.21805164217948914, 0.20879003405570984, 0.03609996661543846, 0.05369060859084129, -0.2319415807723999, -0.05108391493558884, 0.7821439504623413, -0.054207298904657364, 0.6259822845458984, 0.227592334151268, 1.063055396080017, 1.5576226711273193, 0.1953853815793991, 0.8791011571884155, 0.5109755396842957, -0.17158885300159454, 0.62062668800354, -2.0655906200408936, 0.28000327944755554, 0.9986553192138672, -0.26769086718559265, 1.1430137157440186, -1.9113779067993164, 0.3158717751502991, -0.41496148705482483, 1.3988499641418457, 2.526458501815796, -1.3784387111663818, -0.6669036149978638, 0.897443950176239], [0.10008083283901215, 0.8496425151824951, 0.553717315196991, -0.7828408479690552, -0.057483233511447906, 0.07121168076992035, 1.145398736000061, 2.5972514152526855, -0.9914786219596863, -0.9168041348457336, -1.8485780954360962, -0.5215111374855042, 0.8855815529823303, 0.5097020268440247, -0.27411991357803345, -0.8305715918540955, -0.5541777610778809, 0.28117987513542175, 0.5216740369796753, -0.8751375079154968, 0.4830628037452698, 0.8918617367744446, 2.1712615489959717, -0.007227328140288591, -0.4548523724079132, 0.15687531232833862, 0.9998100996017456, -0.8814287185668945, -0.3992522656917572, -1.0106340646743774, -1.1377780437469482, 0.23736324906349182, -1.1831934452056885], [-0.6104848980903625, 1.0857216119766235, -0.9485885500907898, -0.6265377402305603, 0.5829371809959412, -1.1908774375915527, -0.42872679233551025, 1.4106289148330688, -1.3091791868209839, 1.3557720184326172, -2.141005754470825, 0.17049117386341095, -2.7209250926971436, -0.944847822189331, -2.260997772216797, -0.5410262942314148, -0.36211541295051575, -0.5180536508560181, -1.226993203163147, 2.2180254459381104, 0.002996397903189063, 1.2181435823440552, 0.4353761374950409, 0.9949811100959778, 0.6136490702629089, -0.17250528931617737, 0.9570632576942444, -1.1121602058410645, 1.293034553527832, -1.3497583866119385, -2.0008909702301025, -0.672661304473877, -1.2103333473205566], [-0.4312189519405365, 1.1736141443252563, -0.7088649272918701, -1.4104830026626587, 0.4048047959804535, 0.3791087567806244, 0.9302076101303101, 1.415790319442749, 1.008754014968872, -1.404399037361145, -2.126737117767334, -0.12027139216661453, -1.6904947757720947, -0.22009433805942535, 1.1292014122009277, 0.43144530057907104, -0.6649478077888489, -0.8793829679489136, -0.9068319201469421, -0.09543967992067337, 2.093677282333374, -0.16897310316562653, 2.5015931129455566, -0.3262968063354492, -0.613641083240509, -0.9527561068534851, 0.09265468269586563, 0.24771146476268768, -0.9877141714096069, 0.7149574756622314, -1.2323193550109863, -0.769561231136322, -0.9684211015701294], [0.058958500623703, -1.1435996294021606, 0.5013945698738098, 0.4179524779319763, 0.832018256187439, 1.349812388420105, 1.0659297704696655, -0.8787401914596558, -0.22930637001991272, -0.4067283868789673, -0.9028301239013672, -1.1063412427902222, -0.07086167484521866, 0.17364735901355743, -0.06985054165124893, -0.020751772448420525, -0.15894219279289246, -0.656215488910675, 0.3238257169723511, 0.3601861298084259, 0.9660124778747559, 1.2593053579330444, 0.33893126249313354, 0.07528124004602432, -0.35895752906799316, 0.6482630372047424, 1.62595534324646, 2.0863561630249023, -1.0060734748840332, 0.2526039481163025, 2.0944433212280273, 0.3382073640823364, -1.6905033588409424], [0.8251577019691467, 1.2220004796981812, 0.3168874680995941, 0.888359010219574, -0.025898106396198273, -1.4118571281433105, -1.3105484247207642, 0.01634552702307701, 0.3975038230419159, -0.6993120908737183, 0.8043289184570312, -0.22910797595977783, 0.07119172066450119, -0.3453657329082489, -2.2178356647491455, -0.3045235872268677, 1.2358077764511108, 0.1756013184785843, 0.5431310534477234, 0.7098358273506165, 0.6688993573188782, -0.329519122838974, -0.8046314716339111, -1.1520795822143555, -0.23525455594062805, -0.2954461872577667, -0.6903311014175415, 0.3726291060447693, -0.18654386699199677, -0.7467155456542969, 1.0928279161453247, -0.15344937145709991, -0.1993160843849182], [0.8840052485466003, -0.7698653936386108, -0.4522654116153717, -0.07092209905385971, -1.1323481798171997, -0.7357508540153503, 0.07009565830230713, 0.8934064507484436, 0.6738497018814087, 0.9462382793426514, -0.6783565282821655, -0.3028612434864044, 1.841283917427063, -0.7708995938301086, -0.22760140895843506, -1.95283842086792, 0.3485691547393799, 1.664402961730957, -1.0496405363082886, 0.030743686482310295, 0.6442865133285522, -1.1878129243850708, -0.2186037003993988, 1.3686643838882446, 0.460222989320755, 1.2803055047988892, -1.5921156406402588, 1.2887970209121704, -0.2775649428367615, -0.11004466563463211, -1.8101227283477783, 1.5419002771377563, -1.1587024927139282], [0.9291002154350281, 0.084596186876297, 0.2214779108762741, 0.39663711190223694, -0.05439877510070801, 0.6423234939575195, 1.1073285341262817, 1.4806758165359497, 1.051943063735962, 1.0902584791183472, 1.0377099514007568, -0.520418643951416, -0.19684341549873352, -1.4908801317214966, 0.6632447838783264, 0.1158602386713028, 1.4147422313690186, 0.3786199688911438, -1.2686195373535156, -0.7455497980117798, -0.43389302492141724, -0.38863539695739746, 2.125199556350708, -0.33800819516181946, -0.6278405785560608, -1.114946722984314, 2.317918300628662, 1.3987733125686646, 0.4336853325366974, 0.5234782099723816, -1.895303726196289, 0.49250876903533936, -2.2070531845092773], [-1.3285707235336304, 0.6174344420433044, 1.6260950565338135, 0.37963882088661194, -0.029049893841147423, 0.9157254695892334, -0.33024862408638, -1.7232178449630737, 0.13719423115253448, -0.6102094650268555, 1.1938951015472412, 1.722062587738037, -1.1039870977401733, 1.4156923294067383, 0.3075951337814331, -0.47631821036338806, 0.276316374540329, 0.6931729912757874, -0.5518549084663391, -0.049760930240154266, 1.1675598621368408, -0.34996896982192993, 0.1762789636850357, 0.6811070442199707, 0.8087095022201538, -0.3787531554698944, -0.13034191727638245, -0.6533517837524414, 0.6275192499160767, 0.3551935851573944, 0.5595580339431763, -0.09915948659181595, -0.5066484212875366], [0.36866381764411926, -1.452863335609436, -0.35487625002861023, -0.5540529489517212, 1.4882582426071167, -0.34616589546203613, -0.2829681932926178, -1.6756800413131714, -1.0180995464324951, -1.5315579175949097, 1.2502896785736084, -0.29023709893226624, 0.9258211255073547, 0.8707302808761597, 0.12378544360399246, 0.6005163788795471, 0.2953890562057495, -0.5725651979446411, -0.1208818182349205, 1.815057396888733, -0.27903318405151367, 0.5736397504806519, -0.10103701800107956, -0.001651566824875772, -0.5278151631355286, -0.5077694654464722, -0.010868689976632595, -0.862994372844696, -0.04909101128578186, -1.159374475479126, 1.8172900676727295, 0.8774465918540955, 0.26029157638549805], [0.4109805226325989, -0.6665322780609131, -0.2621747851371765, -0.5440706014633179, -1.2828550338745117, -0.520128071308136, -0.5634767413139343, 0.9690923690795898, 1.552543044090271, -0.4168471693992615, -0.29195281863212585, 0.5109667181968689, 0.91270911693573, -0.2447374314069748, -0.25262483954429626, -0.4489617943763733, 0.8128001093864441, -1.2779626846313477, -0.12582305073738098, 0.3173539638519287, -0.4230515956878662, 0.991148054599762, 0.44747430086135864, 0.6123770475387573, -0.9855340719223022, 0.34192579984664917, -0.1518622487783432, -1.0736713409423828, 0.7407025694847107, 0.10420514643192291, -1.9010182619094849, -1.8375543355941772, 1.4476027488708496], [-0.5966198444366455, -0.922839879989624, -0.755737841129303, 0.27998533844947815, 1.3114051818847656, 1.137601375579834, 0.15297336876392365, -0.7651494741439819, 0.08562608063220978, 0.5238931179046631, -1.1408873796463013, -0.1300787329673767, 1.5724623203277588, 0.20016507804393768, -0.49346888065338135, -0.3914109170436859, 0.950228750705719, 0.1795143336057663, -0.3902004659175873, -0.8838879466056824, 0.8075259923934937, -0.05536125972867012, -0.45154261589050293, 0.6876097321510315, 0.6682459115982056, 1.5939826965332031, 2.3425557613372803, -1.4542210102081299, 0.9529045224189758, -0.6169352531433105, 1.0145344734191895, -1.1451705694198608, 0.3621845245361328], [-0.6310398578643799, 1.247787594795227, 0.16208399832248688, 0.5637407302856445, -0.03573311120271683, 1.2982275485992432, -1.2761542797088623, -0.787121057510376, 0.3759695291519165, -1.0352774858474731, -0.9587504863739014, 2.630279064178467, 1.216133713722229, 1.336971402168274, 0.28271642327308655, -0.29102256894111633, -0.7386114597320557, 1.5522829294204712, 1.2330560684204102, -0.40556517243385315, -1.0058372020721436, 0.6582730412483215, 2.0585861206054688, -0.6247584223747253, -0.028852729126811028, -0.8551008105278015, 1.6021804809570312, -0.48001572489738464, -1.1933932304382324, -0.3146092891693115, -0.7777161598205566, -0.06969276070594788, -1.1058052778244019], [0.5849120616912842, -0.24546661972999573, -0.17705562710762024, 1.57626211643219, 0.9599284529685974, -0.3729126453399658, 0.5305541753768921, 0.39738309383392334, 0.3073434829711914, -1.4418699741363525, 0.005576243624091148, -0.49289172887802124, -1.6425397396087646, 0.7578734159469604, 0.33357346057891846, -1.6037713289260864, -1.6080104112625122, 1.4099949598312378, 1.8307626247406006, 0.27382636070251465, -0.6132773160934448, 2.546194553375244, 0.17511484026908875, 1.1581326723098755, -0.3084133565425873, -0.5834526419639587, 0.3963075578212738, -0.6158203482627869, 0.014110096730291843, 0.656326413154602, -0.6901096105575562, -0.20205724239349365, 0.45886942744255066], [-0.11729913204908371, -0.25664034485816956, 0.07495724409818649, 0.45380109548568726, 0.759246289730072, -0.18582025170326233, -1.2590899467468262, -0.25413453578948975, -1.0144968032836914, 0.8736606240272522, 1.2796138525009155, 1.473168969154358, -0.3824344873428345, 0.9264818429946899, -0.17881138622760773, -0.5539810657501221, 1.1899445056915283, 1.824011206626892, 0.28853532671928406, 0.3505304455757141, 0.700546383857727, 0.9234830141067505, -0.8243833184242249, 0.5313417315483093, -0.5177093148231506, 0.283896267414093, -0.28397026658058167, -1.0603071451187134, -1.1193466186523438, -1.1696480512619019, 0.836260199546814, 0.11440522968769073, -0.28174400329589844], [1.271965503692627, 1.6135516166687012, 1.2862478494644165, -0.8334959149360657, -0.08994710445404053, 0.14881105720996857, -1.2271721363067627, -0.09003964811563492, -2.0266544818878174, 0.31303462386131287, 0.08781341463327408, 0.4442133605480194, -0.41720956563949585, -1.3552613258361816, -1.595130443572998, 0.4185713827610016, -0.7423169612884521, 0.9823514819145203, 1.1664539575576782, -0.6256431937217712, 1.0135388374328613, 2.1636769771575928, 0.42244455218315125, 0.15442150831222534, 0.06598708033561707, 0.0008560075657442212, -0.5629571080207825, 0.09078116714954376, -1.5602903366088867, -0.9640783071517944, 0.2809678018093109, 0.12700626254081726, -0.9653075337409973], [-0.41190990805625916, 1.0800275802612305, 1.4719316959381104, 0.5465413928031921, 0.07118783891201019, 0.8857399225234985, -0.4689042568206787, 0.52055424451828, 0.4288977384567261, 0.44225645065307617, 0.085673987865448, -1.9936391115188599, 1.8194899559020996, 2.187563419342041, -1.1808981895446777, -0.4085555374622345, -2.3416576385498047, 0.5441582798957825, -1.2716425657272339, -1.0766119956970215, 0.26422709226608276, -0.3078150749206543, 0.3509288430213928, -0.9602785706520081, -0.18554511666297913, 0.4816834628582001, 0.12471112608909607, 0.1899266391992569, 0.27141115069389343, 0.5790705680847168, 1.9561749696731567, -1.3612653017044067, -0.5675541162490845], [0.24783295392990112, 2.15779709815979, -0.8000459671020508, -0.45192572474479675, -0.09421123564243317, 0.2182575762271881, 0.7175366282463074, 1.077006459236145, -0.42206117510795593, 0.44769203662872314, 1.354027509689331, -1.0458062887191772, 2.3051085472106934, -0.6922785043716431, 0.9222568869590759, 0.6999794840812683, 0.7270565629005432, 0.6835432052612305, -1.9784413576126099, 0.9074832201004028, 0.6455029845237732, -0.775568425655365, -0.7901657223701477, 0.4571259021759033, 1.1505591869354248, -0.8468924164772034, -2.204728364944458, 0.4746527373790741, 1.2476489543914795, -0.5065252184867859, -0.2999795079231262, 0.5412151217460632, -1.0691837072372437], [0.18905626237392426, 0.4585004150867462, 0.28349944949150085, -0.685493528842926, 2.5329437255859375, 0.2978283762931824, 1.4388461112976074, 0.9248319268226624, -1.8097648620605469, -0.02431352809071541, -0.6245569586753845, -0.03504740446805954, -1.065053105354309, 0.24677316844463348, -0.42849695682525635, -0.36154869198799133, -0.42574310302734375, 0.27804142236709595, 0.7193100452423096, -0.14783702790737152, -0.7276104092597961, 0.10737518221139908, 0.9607812762260437, 0.25793495774269104, 0.5576697587966919, 1.0660334825515747, 0.3785721957683563, -2.2036406993865967, -1.3071837425231934, 1.7327054738998413, 0.45204272866249084, 2.5850486755371094, -1.845194935798645], [0.1178782656788826, -0.27912431955337524, -0.5375726819038391, -0.1199905127286911, 0.26637670397758484, -0.3103690445423126, -1.9882277250289917, 0.42506837844848633, 1.7010358572006226, -1.6882364749908447, -2.8391056060791016, 1.1704437732696533, 0.018556518480181694, 1.0918740034103394, -1.233466625213623, 0.559916079044342, -0.06271199136972427, 1.4528725147247314, 0.8983098268508911, -1.3622945547103882, -0.391985684633255, 0.8045783042907715, 0.2922493517398834, 0.4011061489582062, 0.9285716414451599, -0.07386402040719986, -0.6951016187667847, -1.2758677005767822, 1.3758317232131958, -0.5147975087165833, -1.1209068298339844, 0.17347592115402222, -0.35453328490257263], [0.7834960222244263, 0.40997782349586487, 0.11966048181056976, -2.4400768280029297, 0.28203070163726807, -0.19229544699192047, 1.6007441282272339, -0.26282960176467896, -0.16252067685127258, 1.0432004928588867, 0.029711496084928513, -0.815928041934967, -1.2020084857940674, -1.5246762037277222, -1.599250316619873, 0.016343194991350174, 0.6886131167411804, 0.13361197710037231, 0.9387933611869812, -0.2961149215698242, -1.8315678834915161, -1.077594518661499, -0.9028915166854858, -0.15933537483215332, 0.1942756623029709, -0.7943808436393738, 0.5814807415008545, 0.49075937271118164, -0.0647638663649559, -0.43948715925216675, -0.5581042170524597, -0.6745070219039917, -0.15347370505332947], [1.0558977127075195, 1.8703492879867554, -0.8305730223655701, 0.0970921441912651, -0.004852548707276583, -0.5003477334976196, -1.4437106847763062, 0.33082976937294006, -0.5413861870765686, 0.5460694432258606, -0.09412109106779099, 0.15462887287139893, 0.23423166573047638, 0.21196028590202332, 0.20273521542549133, 1.5038715600967407, 0.05774766206741333, -0.30635619163513184, 1.6238412857055664, -1.3871155977249146, 1.398851990699768, 0.12341555953025818, 1.1159530878067017, 0.5320417284965515, -0.31499800086021423, -1.3460302352905273, 0.9976316690444946, -0.6068984866142273, -0.7492480278015137, -1.0540664196014404, -0.809855580329895, -1.4328064918518066, -0.15607419610023499], [-0.12290822714567184, 0.8137884140014648, 0.09228956699371338, 2.197544574737549, -0.4708186089992523, -1.2335220575332642, -0.9762369394302368, 0.3471614718437195, 0.07331334054470062, 1.0429317951202393, -0.7303032875061035, 1.222951889038086, -0.3947540819644928, 0.1705978959798813, 0.12143876403570175, -1.2195340394973755, 1.4548791646957397, -0.7665619850158691, 0.24075686931610107, 1.2966550588607788, -0.5156314373016357, -1.1170153617858887, -0.4083141088485718, -0.912371814250946, 1.5276648998260498, 2.4223849773406982, -2.077627658843994, 0.54134202003479, -1.7901383638381958, 1.0343761444091797, 0.7703152894973755, -1.1080842018127441, -0.4309869706630707], [-0.2424437403678894, 0.30204737186431885, -1.4476678371429443, -0.3034162223339081, -1.2314810752868652, -0.03342253342270851, -1.2494632005691528, 1.7527680397033691, 1.2171679735183716, 0.12748253345489502, -2.1366117000579834, -1.0349247455596924, -0.24897906184196472, 1.0047142505645752, -0.46379679441452026, 0.12867334485054016, -0.17623674869537354, 0.12177765369415283, -0.2108660191297531, -0.8650174736976624, -0.4900001585483551, 1.9130760431289673, -0.2397347390651703, -0.9015310406684875, 0.47260111570358276, -0.2803884744644165, 1.379972219467163, -1.6122297048568726, 0.6353822350502014, -0.1832609325647354, -0.10123685747385025, -0.7042827606201172, -0.15970845520496368], [-0.5523244738578796, 0.9001407027244568, -0.8923831582069397, -0.7077465057373047, -0.48394760489463806, -0.1535741090774536, -0.5188315510749817, 0.982966423034668, -0.2488454133272171, -0.4704359769821167, 0.38203227519989014, -0.870480477809906, 0.07869067788124084, 0.06633273512125015, -0.7788345813751221, -0.25830018520355225, -0.0019482806092128158, -0.6005010008811951, 0.4482612907886505, -1.136610746383667, -1.3467177152633667, -0.3792412281036377, 0.17518313229084015, 0.7861151099205017, 0.3758224546909332, 0.8003153204917908, 0.5923469066619873, 1.9214917421340942, -2.826948404312134, -0.6789836883544922, 1.3999344110488892, -1.86623215675354, 0.11455650627613068], [-0.09975182265043259, 0.89222252368927, 2.4293160438537598, 0.5513182878494263, 0.0403345488011837, 0.29483410716056824, -0.39217403531074524, -0.5658605098724365, 0.13260993361473083, 0.11455827951431274, -1.466707706451416, -0.3683939576148987, -0.5853967070579529, -1.6338467597961426, -0.09513881057500839, 0.46422526240348816, 0.3221718668937683, -0.22149400413036346, 0.3098297715187073, -0.36810991168022156, 0.4308186173439026, -0.19651047885417938, 0.13525915145874023, 1.019740104675293, -0.8474385142326355, 0.3046310245990753, 0.30741381645202637, 0.7362588047981262, 0.3320108652114868, -1.4846196174621582, 1.1359786987304688, 0.27808883786201477, 0.6889991164207458], [-0.6178476214408875, 0.1758618950843811, -0.44059082865715027, 0.3774777054786682, -0.6083014011383057, 0.42484936118125916, 0.19058236479759216, -0.6877420544624329, -0.1411305069923401, -1.3485469818115234, 0.7852031588554382, -1.5798431634902954, 0.3905826807022095, -1.4592013359069824, 1.1005592346191406, 0.638421356678009, -0.8764688968658447, 0.25264671444892883, -0.04195046424865723, -0.07682803273200989, -0.30435478687286377, 0.48943793773651123, -0.8421620726585388, -3.110527753829956, -0.9253372550010681, 0.14670239388942719, 0.7625648379325867, -1.9644103050231934, -0.5531692504882812, -0.4686072766780853, 0.5413572788238525, -1.9815548658370972, -0.5287092328071594], [0.8836514353752136, -0.707846462726593, -0.09792086482048035, -0.9032133221626282, 0.8304860591888428, -1.5760817527770996, 2.1247565746307373, -1.563275933265686, -0.5040923953056335, -1.4268698692321777, 0.43325701355934143, 0.04898202419281006, 0.40175360441207886, -1.1930944919586182, 1.9971994161605835, 0.9700437188148499, 0.08088966459035873, -1.4572614431381226, -0.6745826005935669, 0.760417640209198, -1.3921470642089844, 0.3137053847312927, 0.03265733644366264, 0.04494524374604225, 0.09446583688259125, -2.294584274291992, -0.5220328569412231, -0.11625774204730988, 0.6877982020378113, -0.7935675978660583, 1.8270068168640137, 0.7810477614402771, -0.21742860972881317], [-0.411498486995697, 0.649183988571167, -1.022276520729065, -2.6725966930389404, -0.8240587711334229, -0.5835825800895691, -1.3163875341415405, -2.552618980407715, 0.006152546964585781, 0.6274479031562805, 1.5309202671051025, -0.3272613286972046, 1.6013935804367065, -1.5231143236160278, 0.6702185273170471, -2.0373764038085938, 1.155491590499878, -0.22656221687793732, -0.34705665707588196, 1.7836397886276245, -0.5985383987426758, -1.073401689529419, 0.24577774107456207, 0.3734949827194214, 0.6015621423721313, 0.9095143675804138, 0.24307803809642792, -1.102041482925415, 0.3173125088214874, 2.766350269317627, -0.5821473598480225, -0.6741567850112915, -0.1240425705909729], [-0.4019000828266144, -0.8973191380500793, -0.28295227885246277, 1.0622456073760986, -0.22182512283325195, -0.7337027788162231, 0.1774413138628006, -0.14912033081054688, -0.9055382609367371, 0.9615116715431213, -0.38238903880119324, 1.0749434232711792, -0.9245567321777344, 0.2922643721103668, -0.4102378785610199, 0.5156592130661011, 1.460042953491211, 1.0778677463531494, -0.9597352147102356, 0.3044321537017822, 0.2233821302652359, 1.5301803350448608, 0.4869575500488281, -0.4667167067527771, -1.3828833103179932, -0.7838060259819031, 1.0496279001235962, -1.1216927766799927, -0.8970370888710022, 1.1647794246673584, -1.1396949291229248, 2.6498153602005914e-05, 0.1290542185306549], [-0.2028081864118576, 0.32746559381484985, -0.11991541087627411, -1.0609201192855835, -1.0957273244857788, 0.5834380388259888, 0.03517612814903259, -1.0201400518417358, -1.0790166854858398, -0.6638129949569702, 0.6751697659492493, -0.6744827628135681, -0.4907022714614868, -2.528489828109741, -0.3948071599006653, 1.5093865394592285, 0.8155462741851807, -1.7356312274932861, -2.0590450763702393, -0.8134699463844299, 0.3359469771385193, -1.0745550394058228, 0.6757604479789734, -0.2078760266304016, 1.0201359987258911, 1.8722708225250244, -0.5775936245918274, 0.16357822716236115, -0.5751237869262695, -1.1499483585357666, 1.2616773843765259, 1.018580436706543, 0.025891661643981934], [0.24879558384418488, -0.4312435984611511, 1.0798629522323608, 0.36281508207321167, 0.05852878838777542, -0.22877249121665955, 0.8541343808174133, -0.32099398970603943, 0.043304651975631714, -0.28469425439834595, 1.796641230583191, -1.667542576789856, -0.5478172302246094, -0.13932760059833527, -1.1454373598098755, -1.0049561262130737, 1.9445781707763672, -0.9827422499656677, -1.9929659366607666, -0.47600406408309937, 0.02821497991681099, 0.5002532005310059, 2.1798670291900635, 0.7818178534507751, -0.2828017771244049, -1.3338158130645752, -0.2587030827999115, 0.4407329261302948, 1.322309136390686, 0.722249448299408, -0.63008052110672, -1.0936298370361328, -0.29650503396987915], [0.4591578543186188, -1.0086342096328735, 0.2724023461341858, 1.7054109573364258, -0.0973539873957634, 0.36968323588371277, -0.6030343174934387, 1.3854268789291382, 2.460865020751953, -0.1445964127779007, 0.032494477927684784, -3.14640212059021, 0.19610947370529175, -0.06597701460123062, -0.43496569991111755, -0.6245064735412598, -0.34863030910491943, 2.9437336921691895, 0.07054343074560165, -0.534676730632782, 1.7406318187713623, -0.22371307015419006, -1.1075114011764526, -0.4121533930301666, 0.12121990323066711, 0.5239804983139038, 0.09753231704235077, 0.6692305207252502, 0.3961004316806793, 0.30547258257865906, -1.1276769638061523, 0.7827844619750977, -0.9659058451652527], [-0.37735453248023987, 1.1319702863693237, -0.8955926895141602, -0.022732237353920937, 2.113635540008545, 1.9208505153656006, -1.4199272394180298, -0.8610394597053528, 0.9025363922119141, 0.5706536173820496, -1.3897463083267212, 0.2265298068523407, 0.6204609870910645, -2.151845932006836, -0.3657706677913666, 0.7708147168159485, -1.5029255151748657, -0.9596942067146301, 1.4276182651519775, -0.7302789092063904, -0.44149041175842285, -1.7087781429290771, 0.42269179224967957, -0.13032133877277374, 0.38035231828689575, -1.8845676183700562, -0.7474175691604614, 0.15753771364688873, 1.2898565530776978, 0.8266034722328186, -0.3493284583091736, -0.39641836285591125, 1.2327839136123657], [1.3476006984710693, -1.57535719871521, 0.8311913013458252, -0.2570064067840576, 0.27345404028892517, -0.5667861700057983, -0.7766721248626709, 0.5432957410812378, 0.036060675978660583, -1.622856616973877, -1.0856451988220215, 0.7108325958251953, -0.18267357349395752, 0.2656411826610565, -0.9680434465408325, 0.0653529241681099, 1.0528850555419922, 1.2031097412109375, -0.6813675761222839, 0.394542396068573, 0.51191246509552, -0.8838182091712952, 0.29558685421943665, -0.653175950050354, 0.4141930341720581, 1.453726887702942, 0.07392729073762894, 0.5887890458106995, -3.196909189224243, 0.7179835438728333, 2.036597967147827, -0.7475149035453796, -0.34732139110565186], [-0.7521265149116516, 0.7902291417121887, 0.8115870356559753, 0.02020699344575405, -0.7569093108177185, -0.04431403428316116, 1.6813181638717651, -1.856430172920227, -1.4353325366973877, 0.07819728553295135, -0.4952496290206909, 1.6870297193527222, -0.99561607837677, 1.5422470569610596, -0.6081114411354065, 0.14448733627796173, 1.0878857374191284, -0.46932536363601685, -1.785456895828247, 0.6781886219978333, 0.38084152340888977, -2.3060057163238525, 1.1554298400878906, 0.47561678290367126, 0.78516685962677, -1.2904921770095825, 0.7216408252716064, -0.32067495584487915, 1.8929240703582764, -0.5438562631607056, 0.41665858030319214, 1.535969614982605, 0.2279064804315567], [0.15535961091518402, 0.7406030893325806, -1.463098168373108, -0.6895695924758911, -1.1350010633468628, 0.6944214701652527, -0.9599978923797607, 0.2859623432159424, 1.1239513158798218, -2.448625326156616, -0.057534534484148026, -0.28313711285591125, 0.17940330505371094, -0.9996035099029541, -0.9138743281364441, 0.283552885055542, -0.726229190826416, -1.4106180667877197, 0.8675287365913391, 0.6637020707130432, 0.10359765589237213, 0.3666832149028778, 2.4345405101776123, 0.3583659529685974, 0.5635316967964172, 0.24125276505947113, -1.7587363719940186, -1.881188988685608, 3.0941736698150635, 0.023262597620487213, -1.0011012554168701, -0.3975231647491455, -1.452553153038025], [-1.1849809885025024, 1.5810699462890625, -0.20714430510997772, 0.3921959698200226, -0.21660295128822327, 1.4103177785873413, -0.7912853956222534, -0.19688163697719574, -1.4309048652648926, 0.3428462743759155, -1.0063457489013672, -2.0523338317871094, -0.09952462464570999, 1.5888631343841553, 0.3686487376689911, 1.3976025581359863, -0.9950752854347229, 1.6285934448242188, -1.731810450553894, -0.21845823526382446, 1.3074182271957397, 0.44160354137420654, 0.19180546700954437, 0.8759381771087646, 0.2426181137561798, -0.4321993589401245, -1.1400668621063232, -0.3929051160812378, -1.392181396484375, 1.501660943031311, -0.8322390913963318, 1.2833006381988525, -0.6844863295555115], [-0.6672040224075317, 1.0179283618927002, 0.3054528832435608, -0.3837234377861023, 1.4080519676208496, -0.40565025806427, 1.1819453239440918, 1.1477830410003662, -0.9227980971336365, 1.2134389877319336, 0.10512339323759079, 0.5907616019248962, 0.16117775440216064, -0.17673783004283905, 0.91776043176651, -1.6601932048797607, 1.0091304779052734, -0.14972969889640808, 0.07644187659025192, 0.3250148296356201, 0.7480509281158447, -0.4070965647697449, 0.14151960611343384, 0.16359491646289825, 0.20069973170757294, -0.5770962834358215, -1.1982100009918213, 0.6514677405357361, 1.1762988567352295, 0.9312636256217957, 0.876453697681427, -1.5567858219146729, -2.3442628383636475], [-1.342703938484192, -0.2531694173812866, 1.2011736631393433, -0.8122031092643738, -0.3437148630619049, 1.7216721773147583, 0.5604905486106873, 0.15356692671775818, 1.4241329431533813, 1.660108208656311, 0.1539609283208847, 0.6394777297973633, 0.27382105588912964, -1.464964509010315, 0.29617181420326233, -1.0867019891738892, 1.7562514543533325, 1.6548001766204834, -0.978439211845398, -0.04250185564160347, 0.5782598257064819, 0.7043735384941101, 0.4897703528404236, -0.9250810146331787, -0.2814149856567383, 0.6922146081924438, 0.916297972202301, 1.8622748851776123, -0.09200119972229004, -0.533656895160675, -0.4096255898475647, -0.6955307722091675, 0.7975021600723267], [-0.5455357432365417, -0.6299567222595215, 0.40131863951683044, -0.2015092968940735, -0.2846023440361023, 0.3895130157470703, 1.4465711116790771, 0.46364831924438477, 1.2390553951263428, 0.17226693034172058, 0.8148647546768188, 1.0537219047546387, -0.279084712266922, -0.2581466734409332, -0.3699871599674225, 0.2878666818141937, -1.1022101640701294, 0.778631865978241, 0.2288026362657547, 0.13576599955558777, -0.42807817459106445, -0.735599935054779, -0.22649043798446655, 1.7971843481063843, -0.2375725358724594, -1.1568461656570435, 0.18591952323913574, 2.6046440601348877, 0.1627299189567566, 0.944288432598114, 1.3004252910614014, -0.8163477182388306, 0.5591328740119934], [-0.4709741175174713, 0.5736753344535828, 0.1597989797592163, -0.9958658218383789, -0.052086565643548965, -0.5885547399520874, 0.06985108554363251, -1.2645072937011719, 0.24457725882530212, 1.5869945287704468, -0.26221880316734314, 0.9368643760681152, 1.8777137994766235, -0.004166303668171167, 1.1806583404541016, -1.057949185371399, -0.8119271993637085, 0.06859863549470901, 1.0321862697601318, -0.28667089343070984, 0.8221667408943176, 1.4592416286468506, 0.010154569521546364, 1.2899341583251953, -1.6832224130630493, 0.5290600657463074, -1.0123006105422974, 1.2305450439453125, -0.5603728890419006, 2.1405794620513916, -1.505128026008606, 1.7054356336593628, -0.5407900810241699], [0.655705451965332, 1.5108191967010498, -0.9189920425415039, -0.1964569389820099, -0.17524243891239166, -1.4009865522384644, 1.9629055261611938, -0.9305647611618042, 1.2636094093322754, 0.5956193208694458, -0.6341479420661926, -0.4355776309967041, 0.9813247919082642, -0.14396695792675018, -0.23107609152793884, 0.2965072989463806, -0.5305481553077698, 0.7657456994056702, 1.2227909564971924, 0.8565900921821594, 1.2214899063110352, -1.7086846828460693, 1.1461412906646729, 1.024808645248413, 1.3557041883468628, -0.6942174434661865, -0.790787935256958, 0.7096025943756104, 0.009567048400640488, -0.043766796588897705, -1.7742635011672974, 0.596713662147522, 0.7206476330757141], [1.1110846996307373, 1.4418528079986572, 0.5501821041107178, -0.012243089266121387, 0.07164955884218216, -0.598445475101471, 0.05818641185760498, 0.8833771347999573, 0.9325059056282043, 1.005613923072815, -0.26764312386512756, 0.7496811151504517, -2.223517417907715, -0.6008679270744324, 0.049457889050245285, -1.2473622560501099, 1.5244554281234741, -1.1402913331985474, 0.8167306184768677, 0.42594051361083984, -0.930418074131012, -0.008224713616073132, -2.437913179397583, 2.0495619773864746, 0.6611108183860779, -0.5753529667854309, -0.1084311306476593, 0.40556231141090393, -0.016497598960995674, 0.42613473534584045, -1.177101969718933, 1.1854827404022217, 1.3865575790405273], [0.6715707778930664, 0.23678526282310486, -0.5845908522605896, -1.029527187347412, 1.4926306009292603, 0.1657276302576065, 1.5372154712677002, -1.08051335811615, -0.45544275641441345, -0.4187227785587311, 0.5588340163230896, -0.5487812161445618, -0.9059581756591797, 0.6148850917816162, -0.1524791419506073, 1.8389335870742798, 0.8743226528167725, -1.7042417526245117, -0.11361684650182724, -0.39058685302734375, -1.6960951089859009, 2.2449004650115967, 0.41242319345474243, 0.9119999408721924, -1.5481473207473755, -0.2071159929037094, 3.003772735595703, 0.228903129696846, 1.091536521911621, 0.6539292931556702, -0.4499324858188629, 2.986365556716919, -0.5127891898155212], [-0.2718610465526581, 0.16418641805648804, 0.5095974802970886, 0.5097606778144836, -0.3975328207015991, -0.7058295011520386, -1.1237707138061523, -1.0809450149536133, 1.581896185874939, 0.07424977421760559, -2.8032686710357666, -0.24111449718475342, -0.19439703226089478, -0.3614363372325897, -0.2929310202598572, -0.10639752447605133, 0.9484248161315918, -0.32638198137283325, -0.6044294238090515, -0.358387291431427, 0.9647265076637268, -0.4139673709869385, 0.6411058902740479, -0.13592977821826935, -0.5473378896713257, -0.5925225615501404, -0.1532672792673111, 0.48143720626831055, -0.25985896587371826, 1.2778815031051636, -0.24762214720249176, 2.2479653358459473, 0.6138080954551697], [1.0104620456695557, 0.9764436483383179, 0.48834970593452454, 0.04243963956832886, 0.776884138584137, 0.3243432641029358, 0.0672796294093132, 0.9662261605262756, -0.36475735902786255, -0.4502299129962921, 1.4984616041183472, -1.9490631818771362, 0.08436036854982376, 1.8433758020401, -0.7476325035095215, 1.2463141679763794, -1.4460396766662598, 0.6630874872207642, 0.46807950735092163, -0.30651599168777466, -0.4116362929344177, -1.4919322729110718, 0.22375735640525818, 1.931197166442871, 0.3064797520637512, -0.8897575736045837, -0.5654723048210144, -0.3356829285621643, 0.697172224521637, -0.5634943842887878, -0.8153116703033447, -0.010205646045506, 0.444952130317688], [0.5379674434661865, -0.10962388664484024, 0.7955824136734009, -0.7129462957382202, 1.1925667524337769, 0.590896487236023, -0.11140540987253189, 0.4332324266433716, -0.7483990788459778, -1.2484228610992432, -0.12252449989318848, -0.15628136694431305, 1.6887240409851074, 0.6766747236251831, 1.2852948904037476, 0.6086188554763794, -0.9589383006095886, 1.4510873556137085, -1.1004343032836914, 0.9059833288192749, -0.5025708675384521, -0.9503452181816101, -1.18232262134552, -0.9117202162742615, -1.3863508701324463, 0.9951692223548889, 0.19955621659755707, 0.29960504174232483, 0.5199938416481018, 1.7268478870391846, -0.6204413771629333, -1.247615933418274, -1.7363168001174927], [0.6349338889122009, -0.10219475626945496, 0.6096503138542175, 0.3151339292526245, 0.17595121264457703, 2.189389944076538, -0.45232319831848145, 0.5709381699562073, -0.5753778219223022, -0.9573590755462646, 1.2233507633209229, 0.4391709268093109, 1.0139927864074707, -1.169247031211853, -0.8621594905853271, -0.43977394700050354, -0.17959201335906982, 1.2750533819198608, 0.16119475662708282, -0.5230388641357422, 0.9897867441177368, -0.79015052318573, 0.421845942735672, -1.4592516422271729, 0.8351548314094543, 0.6007422804832458, -0.8383752703666687, 0.43118295073509216, 0.017992477864027023, 0.3099895417690277, -1.9171481132507324, -0.45099741220474243, -0.6446613669395447], [0.3962700366973877, -1.340181827545166, 0.3382468819618225, 1.3317980766296387, 0.6941142678260803, 0.33498814702033997, 0.023164577782154083, 1.5451350212097168, 0.1654238998889923, 0.9748136401176453, -0.8701626062393188, -1.3598109483718872, -0.8989429473876953, 0.16383272409439087, 1.2175190448760986, 0.25337010622024536, 0.0445072203874588, 3.1367127895355225, 0.05341050401329994, 0.46899038553237915, 0.7140439748764038, 0.2330266833305359, -1.8379786014556885, -0.4849400222301483, -0.388474702835083, -0.8241013288497925, -0.45045924186706543, -0.20321278274059296, 0.7179040908813477, 0.9851860404014587, -0.3603048622608185, 0.29468128085136414, 0.49047017097473145], [0.8072888255119324, 0.8501538634300232, 0.49563121795654297, -0.7706298828125, -0.24841029942035675, 0.2658052146434784, 0.9102709889411926, 0.1642790585756302, -0.07221987843513489, -0.36116504669189453, 0.15112070739269257, -0.7946844100952148, -1.4261233806610107, -0.3537500500679016, -1.2525705099105835, 0.3894680142402649, 1.1545437574386597, -0.05568534880876541, -0.8082097768783569, -0.2902841567993164, -0.5998721122741699, 0.9782031178474426, -0.9392404556274414, -0.4774712026119232, -0.6799189448356628, 0.7467394471168518, 0.9255792498588562, 0.9164984822273254, -0.33755922317504883, -0.21974721550941467, -0.21023720502853394, 0.4549408257007599, 0.2377631813287735], [-0.5825505256652832, 1.076512336730957, 1.307955026626587, 1.006879210472107, 0.897347092628479, 0.6963385343551636, 0.6140161156654358, -1.3824349641799927, -0.05971363186836243, -0.10898933559656143, -0.6506983637809753, -0.3349820673465729, 0.7221653461456299, 0.14076140522956848, 1.809438943862915, -1.0431187152862549, 1.5451257228851318, 0.6420555114746094, 0.9097983837127686, -0.4971393644809723, 0.5872090458869934, -0.7133561372756958, 1.441153645515442, -1.661612868309021, -0.018826572224497795, 0.015916980803012848, -1.1634621620178223, -1.0840901136398315, -2.08380389213562, -0.16182401776313782, 0.23829607665538788, 0.3007655739784241, -1.3186943531036377], [-0.44213423132896423, -0.037892162799835205, -0.29497313499450684, -0.10828397423028946, -0.4590452015399933, 2.256366014480591, 0.4609198570251465, 1.3948780298233032, -0.21686498820781708, -0.5338066816329956, -0.19558311998844147, -1.1505666971206665, -1.6916252374649048, 1.034031867980957, 0.2578372359275818, -1.0571966171264648, -1.3068194389343262, -0.08144689351320267, -0.5049028396606445, -0.9283003211021423, -0.9103439450263977, 0.14212413132190704, -0.5547571182250977, 1.0003418922424316, -1.349346399307251, 1.3812366724014282, -1.455902099609375, 0.9209030270576477, 1.0107460021972656, 0.5485751032829285, -1.723645567893982, 1.6945143938064575, 0.7284957766532898], [-0.6932163834571838, -0.10786975920200348, 0.11425288021564484, -0.8162112832069397, -1.0138232707977295, -0.05089575797319412, 2.2441375255584717, 1.9320265054702759, 1.9377390146255493, 0.7805801630020142, 1.223371148109436, -0.8084906339645386, -0.09585051983594894, -0.29309436678886414, 0.3153012990951538, 0.04827297851443291, 0.45888346433639526, 1.1791728734970093, -0.03577391058206558, 0.15862725675106049, 0.2500132918357849, 0.5550158023834229, 0.02341964840888977, -0.5135756731033325, -0.31328558921813965, -1.1082072257995605, -0.11922600865364075, 1.3602843284606934, 0.5340197086334229, -0.6903083920478821, 0.21480897068977356, 2.2926814556121826, -0.7177032828330994], [-0.0002914220676757395, -0.5946378111839294, -0.30292361974716187, -0.4385177195072174, 0.020181704312562943, 0.7125278115272522, 0.26379770040512085, -1.0902303457260132, 0.2784912884235382, 0.9155689477920532, 0.8632775545120239, 1.2844743728637695, 0.4585121273994446, -0.045723944902420044, -0.10946491360664368, -2.127028703689575, -0.7962614297866821, 0.4221169948577881, 1.7106565237045288, 1.522571086883545, 1.47812819480896, 1.7274001836776733, 0.013088109903037548, -0.4605506956577301, -0.0469241663813591, 0.15933215618133545, 1.1901718378067017, -0.27606287598609924, -0.12611736357212067, -0.46286633610725403, -0.4945366084575653, 1.8050413131713867, -0.8965598940849304], [0.46353283524513245, 0.8118410110473633, 1.5474640130996704, -1.6362894773483276, 1.6403263807296753, -0.0931326225399971, 0.03012263774871826, 0.37400296330451965, -0.6522873640060425, 1.8410462141036987, -0.160340815782547, -1.7397395372390747, -0.8487356901168823, -1.3986855745315552, -1.4567527770996094, 1.8844343423843384, 1.7096952199935913, -0.057173874229192734, -0.1916111707687378, -0.048135485500097275, 0.33210429549217224, -0.0011667202925309539, -0.9291170835494995, 0.9142212271690369, -1.1125553846359253, 0.8080406188964844, 0.5250917673110962, -1.1230629682540894, -1.2298368215560913, 1.4291093349456787, 0.17084479331970215, 0.4628541171550751, -0.37738654017448425], [-1.7168173789978027, -0.79679936170578, 0.47438952326774597, -0.03323895111680031, -2.200946092605591, 0.28903624415397644, 0.11171242594718933, -0.9153061509132385, -1.205307126045227, 0.8588343262672424, -1.1968153715133667, -0.22929485142230988, -1.7436155080795288, 0.48409542441368103, -0.19750800728797913, 1.125417709350586, -1.8437316417694092, -1.1578319072723389, 0.830960750579834, 0.03295467793941498, -0.9102604985237122, 0.2295677810907364, -0.7631882429122925, 1.5859051942825317, 1.7647831439971924, -0.2606774568557739, -0.8691351413726807, 0.6395530104637146, -0.3670071065425873, 1.2127323150634766, -1.384131908416748, -1.6576457023620605, 0.9315999150276184], [-0.1901092678308487, 0.15250833332538605, 1.4767608642578125, 0.8225291967391968, -0.7374611496925354, 0.5035212635993958, 0.16792108118534088, -1.110031247138977, 0.8978986144065857, 2.2789857387542725, -0.01859212853014469, 0.23913635313510895, -0.9899616241455078, 1.6233699321746826, -0.7352895140647888, 0.19514091312885284, 2.2854528427124023, 0.3905198872089386, 0.05282950773835182, -1.0052917003631592, 0.3071788251399994, 0.0218991469591856, 1.7466535568237305, 0.0765717625617981, -1.0938040018081665, 1.487621784210205, 0.9654042720794678, -2.1787402629852295, 0.11251184344291687, -0.3669669032096863, -0.43959495425224304, -0.9711886644363403, -0.2653578221797943], [0.4883911609649658, -0.1366814523935318, -0.22245627641677856, 1.492628812789917, -0.5000953078269958, 0.011346436105668545, -2.027451276779175, 0.4620661437511444, 1.2077507972717285, -1.1447011232376099, -0.6947562098503113, -1.560494303703308, 0.8758922815322876, 0.9734538197517395, 1.0683873891830444, -0.08005891740322113, 0.8937070369720459, -1.2804558277130127, 1.8618625402450562, -2.190490961074829, 0.7420463562011719, -0.8766722083091736, -1.3853181600570679, -0.6255247592926025, -1.386794090270996, 1.0078984498977661, -0.8668960332870483, -0.516183078289032, -0.18681392073631287, 0.0008066120208241045, 0.9952163696289062, -0.24916338920593262, 2.110290288925171], [-0.6123496294021606, -0.79093337059021, 0.25535672903060913, -0.6212890148162842, 0.1793450266122818, 0.8998268246650696, -0.141807422041893, -2.6670420169830322, -0.6959421038627625, 0.1462794989347458, 2.0449576377868652, 0.3814696967601776, -0.6433500051498413, -1.7989311218261719, 0.6576605439186096, -0.8158069252967834, -0.6974913477897644, 1.2963255643844604, 0.10407261550426483, -1.1250888109207153, 1.2866618633270264, -1.0405330657958984, 0.8193052411079407, 1.5314353704452515, -0.8328204154968262, 0.046935439109802246, -0.9499310255050659, 1.2585231065750122, 0.37189844250679016, -0.38841551542282104, -0.3072735369205475, -0.5726585984230042, -1.8815550804138184], [-1.1282522678375244, 2.338289737701416, -1.0639710426330566, -0.9025765657424927, -0.7640727758407593, -0.8360024094581604, 1.2497981786727905, -0.1505328118801117, 0.21464863419532776, 0.7032263875007629, 0.7288225889205933, 1.3807239532470703, -1.6358087062835693, -1.3025013208389282, 1.9142838716506958, -0.07141441851854324, -0.9467923641204834, -0.8982713222503662, -0.24560551345348358, -1.3502590656280518, 0.2993324100971222, 0.1274292916059494, 2.2071425914764404, -1.2708265781402588, -1.736875295639038, 0.7527617812156677, 1.166516661643982, -1.235296607017517, -0.5670072436332703, -1.4617037773132324, -0.5044524073600769, 0.004987749736756086, -0.4064129889011383], [1.1510289907455444, 2.1280131340026855, -0.37468990683555603, -1.0767215490341187, 0.6087378859519958, -0.17871983349323273, -1.2863215208053589, -0.24449016153812408, 1.2206354141235352, 0.12747247517108917, 0.7038969993591309, -0.4162095785140991, 0.8211551904678345, -1.1012555360794067, 0.43024560809135437, -0.42478734254837036, 0.13249751925468445, 0.3793390393257141, 0.5576367974281311, 1.1176685094833374, 0.9232136011123657, -2.2725038528442383, 2.2043395042419434, 0.14422918856143951, -0.09261665493249893, -0.4099145531654358, -2.3057503700256348, -1.9185155630111694, 0.2448372095823288, -0.34202584624290466, 1.6852972507476807, -0.5152315497398376, -0.06475534290075302], [0.6277225017547607, 1.343847393989563, 0.9057518839836121, -0.9130347371101379, 0.6935823559761047, 0.22608888149261475, -0.27401137351989746, -0.20973306894302368, -0.008544979616999626, 0.5475669503211975, -0.4169919192790985, 0.4835258424282074, 1.0624033212661743, -0.4359672963619232, 1.5791866779327393, 0.024219442158937454, 1.768108606338501, 0.15865063667297363, 1.1534420251846313, 0.24564626812934875, -0.23735898733139038, 0.6403951644897461, -2.3970699310302734, -0.10857138782739639, -1.175552487373352, 0.1877763867378235, -1.7277096509933472, -0.5606825351715088, 1.2328656911849976, 0.08057213574647903, -1.6007781028747559, -2.0569236278533936, 0.4287566840648651], [-0.3781494200229645, 0.08302357792854309, -1.4343997240066528, 0.035304222255945206, -0.5105542540550232, 2.1473026275634766, 1.374839425086975, -0.3797990083694458, -0.6943986415863037, 0.688265323638916, -1.322542667388916, 0.5856281518936157, -0.6021785736083984, 1.4068803787231445, 0.9019030928611755, -1.4543474912643433, -0.6183632612228394, -0.6414461731910706, 2.2792935371398926, -1.8074005842208862, 0.7496793866157532, 0.41809871792793274, 0.3843173682689667, 0.10710449516773224, -1.4421024322509766, -0.2606712281703949, -0.7239384055137634, 0.1345309168100357, 1.0594650506973267, 0.1424323320388794, 0.3430371880531311, -0.49805089831352234, -1.506876826286316], [0.5536576509475708, -0.13880565762519836, -0.31217190623283386, -0.09218280017375946, 1.0231101512908936, -0.6141909956932068, -1.409912109375, 0.3822123110294342, -1.0351194143295288, -0.6635105013847351, 0.7748063802719116, -0.04629490524530411, -0.3634531497955322, -1.4182227849960327, 0.6587855815887451, 0.6942228078842163, -0.4391571879386902, -0.18871107697486877, -0.17769551277160645, -0.06762415915727615, 0.082160085439682, -0.1623288244009018, 1.3438297510147095, -0.5038801431655884, 0.7746881246566772, -1.1341605186462402, -0.5643510818481445, -0.8499816656112671, -0.9629607200622559, 0.787268877029419, -0.25150325894355774, 0.03418194130063057, 0.3707565367221832], [2.211822509765625, -0.17215290665626526, -0.6628847718238831, 0.3114860951900482, 0.7395480275154114, 3.2560267448425293, -0.9368777275085449, -0.40577441453933716, 0.7134570479393005, 0.29943713545799255, 0.9271292090415955, 1.1843352317810059, -0.26603496074676514, 0.06675112247467041, -1.3493189811706543, -0.5748262405395508, 0.6887281537055969, 0.8366774320602417, -0.30344390869140625, 0.25536638498306274, -0.31504952907562256, 2.01041841506958, -0.652396023273468, -1.3710416555404663, 1.45658278465271, 0.26439526677131653, 1.1204167604446411, -0.8518632054328918, 0.16138184070587158, 0.35388514399528503, -0.4055531322956085, 0.36263737082481384, 0.5138844847679138], [0.5067747235298157, 0.376921683549881, -2.1226162910461426, 0.22110901772975922, -1.954450011253357, -0.2685053050518036, -0.6590883731842041, 0.9730069637298584, -1.5004736185073853, 0.064193494617939, -0.29376959800720215, -0.12236402928829193, -0.20160993933677673, -0.21976657211780548, 0.05558785796165466, 1.2460516691207886, 0.2034911811351776, 1.0063658952713013, -0.4085617959499359, 2.334580898284912, -0.9270104765892029, -0.07431195676326752, -0.0287432000041008, 0.2907531261444092, 0.4318520724773407, 1.039694905281067, -0.10495354980230331, 0.7860361337661743, -0.21998649835586548, -0.13772328197956085, -0.0657876580953598, 0.143443301320076, 0.47952812910079956], [0.8853092193603516, -0.6280930042266846, 0.46765556931495667, 0.15162965655326843, 0.4601060152053833, 0.7201613187789917, 0.7946559190750122, 1.1804063320159912, -1.0738109350204468, -0.3237019181251526, -1.3075422048568726, 0.7624930143356323, 1.7441972494125366, 0.8447619676589966, -0.043515898287296295, 0.13545282185077667, -0.638297975063324, -1.8408278226852417, -1.470394253730774, 0.676672101020813, 0.14562706649303436, 1.1913836002349854, 0.9570869207382202, 2.5883615016937256, 0.7343138456344604, -0.09338003396987915, -1.508283019065857, -0.18104489147663116, 0.5343144536018372, -0.2588661015033722, -0.7898306846618652, 2.006159543991089, 0.668522834777832], [-1.5077425241470337, 0.17905917763710022, 0.10039761662483215, 1.0174533128738403, 3.186098098754883, 0.09975297003984451, 1.6433402299880981, 0.47086581587791443, 0.12995728850364685, 1.9279919862747192, 3.6413018703460693, 0.31131911277770996, 2.1361563205718994, -2.0883195400238037, -0.5657602548599243, 0.9020889401435852, -0.8399468660354614, 1.8687858581542969, 0.6193453073501587, -0.3669034242630005, -0.6407334208488464, -0.5808148980140686, -0.05950500816106796, 1.667920470237732, -1.4498836994171143, -1.5542470216751099, -0.3856695294380188, 0.6164229512214661, -0.7595694065093994, 2.2364389896392822, 0.467293918132782, 1.470300555229187, 0.6347008943557739], [-1.4564130306243896, -0.6765992641448975, 0.3615715205669403, -0.5281131267547607, 0.6968165636062622, 1.488263726234436, 0.1686650812625885, 0.28559610247612, -0.049785975366830826, 0.8563931584358215, -0.041307877749204636, 1.814937710762024, -0.1939123421907425, 0.09572228044271469, 0.5790285468101501, 0.1629927009344101, 0.4066304862499237, 1.0039538145065308, 1.2486203908920288, -1.6940885782241821, -1.0968350172042847, -0.4258388578891754, 0.48954182863235474, 0.9373023509979248, -0.7806645631790161, -2.740105628967285, -0.8407992124557495, 0.15679121017456055, 0.8107839226722717, 0.3368450105190277, 1.4219268560409546, 0.07856329530477524, -1.5182236433029175], [0.45420488715171814, -0.9497618079185486, 0.92164546251297, -0.22306755185127258, 0.1111341044306755, -0.028055664151906967, 0.7263930439949036, 0.24287785589694977, -0.8738806247711182, -0.7169997096061707, -1.7889646291732788, -1.2343329191207886, 1.3977442979812622, 1.2060128450393677, 0.7650768160820007, -2.010441303253174, 0.13205067813396454, -0.42209574580192566, -0.6646576523780823, -1.2238878011703491, -0.713503360748291, -0.3271903693675995, -0.3871009349822998, -1.4127706289291382, -0.6458166241645813, -0.43636682629585266, 0.06816619634628296, -0.4804365634918213, -0.1337546706199646, -1.5347882509231567, -0.9429185390472412, -0.22242207825183868, -0.6839048862457275], [1.6127499341964722, -2.1301257610321045, -0.2941964268684387, 1.0255056619644165, 1.8213095664978027, 1.0505739450454712, -0.14123818278312683, -0.11628241091966629, -1.167693853378296, 0.5351464748382568, 0.8482081294059753, -2.0974090099334717, 0.13555806875228882, -0.8512662649154663, 1.5839051008224487, -0.6690922379493713, -1.1596641540527344, 1.9055649042129517, 0.3420248031616211, 0.34763485193252563, 0.12533119320869446, 0.26697900891304016, -0.2543436288833618, 0.08718355745077133, 0.940015971660614, 0.5292513966560364, -0.3284715414047241, -0.9153077006340027, -0.08015144616365433, 1.698681116104126, 1.5306049585342407, -0.5898084044456482, -0.19488120079040527], [0.7260183095932007, -1.528337836265564, -1.1739665269851685, -0.2574179470539093, 0.13622625172138214, 0.17208577692508698, -1.3026124238967896, 0.5990827083587646, 0.6075469851493835, 0.940990149974823, 0.39399489760398865, -0.7671218514442444, 0.8361636400222778, 0.20514416694641113, 1.1771814823150635, 0.12926557660102844, 0.30112385749816895, 1.0858094692230225, 0.18056470155715942, 1.005041480064392, -0.11972194910049438, -0.1586945354938507, -1.0431642532348633, 0.09372267872095108, -1.5453323125839233, -0.3103080987930298, 0.7929263710975647, 0.5885566473007202, 0.6833533644676208, 0.4885355532169342, -1.377888798713684, 1.3206868171691895, -0.5429345965385437], [0.7086204290390015, -0.15579718351364136, -0.372831255197525, -0.26988351345062256, 0.931614100933075, 2.1314213275909424, -1.453754186630249, -1.2828974723815918, 0.9007941484451294, -0.6948462128639221, 0.8732581734657288, -0.6015904545783997, 0.2550724744796753, -1.7074774503707886, -0.7027968764305115, -0.12872253358364105, -1.5021085739135742, -0.03226076439023018, -0.10156629979610443, 0.9753323197364807, 0.25750234723091125, -0.3468782603740692, -0.8774768114089966, 0.5721048712730408, 0.24697577953338623, -1.3343788385391235, -1.1777393817901611, -1.2814688682556152, -0.03756767883896828, -0.3197539448738098, -0.6632862687110901, 1.5130659341812134, -0.15769442915916443], [-0.5045366883277893, 0.4303235709667206, 0.33166995644569397, -0.5637401342391968, -0.493863046169281, -0.5936495065689087, -0.3907798230648041, -1.5139704942703247, -0.2879766821861267, -1.0860978364944458, 1.210008978843689, -1.138954758644104, -0.07679957151412964, 0.49157842993736267, 1.2073862552642822, 2.647233247756958, -1.9453431367874146, 1.6647695302963257, -0.588130533695221, 1.9130196571350098, -2.5753021240234375, -1.6688189506530762, -1.0997759103775024, 0.21720866858959198, -0.05055706575512886, -0.13173693418502808, -0.5104259848594666, -1.3219358921051025, -0.08442071080207825, 2.759702682495117, 0.6147616505622864, -2.0467963218688965, 0.5150989294052124], [0.09250544756650925, -1.2706100940704346, 0.06536448001861572, 0.17193709313869476, 0.6890925168991089, -0.8056824803352356, -0.45835110545158386, 1.0013201236724854, 2.128859519958496, -1.5625808238983154, -0.6142633557319641, 0.7640143632888794, 0.1901247650384903, -0.7514694333076477, -1.3824840784072876, -1.3122034072875977, 0.18203021585941315, 0.2958204448223114, -0.5488116145133972, 0.36474141478538513, 0.7060771584510803, 0.8159492611885071, 0.10324668884277344, -0.4647735357284546, -0.6407670378684998, 0.7723807692527771, 1.5966094732284546, -0.4363902509212494, -0.4286845326423645, -0.30450043082237244, -1.8766870498657227, 1.548869013786316, -0.3808154761791229], [0.6174956560134888, -0.35509178042411804, 0.016864269971847534, 1.4150803089141846, 0.6854870915412903, 1.7020164728164673, -1.0543280839920044, -0.5933516025543213, 1.0368191003799438, 1.3527263402938843, -0.9261382818222046, 0.727880597114563, 0.12169080227613449, 1.314273715019226, -0.2755199372768402, 1.1199404001235962, 1.5523991584777832, 1.0263627767562866, 2.2440619468688965, 3.0737438201904297, -0.9042323231697083, -0.3360849916934967, -0.965178370475769, -2.3605213165283203, 0.08635759353637695, 1.3465516567230225, -0.2001948207616806, 0.30899375677108765, -0.8944666981697083, 0.8267188668251038, -0.24010950326919556, -0.2792750597000122, 1.786444902420044], [-1.211346983909607, -0.19120876491069794, -1.5953195095062256, 0.41403791308403015, -0.06878161430358887, -0.5102267861366272, -0.10810979455709457, 1.4759727716445923, -0.3368586301803589, -0.895158588886261, 1.1023606061935425, 1.5077314376831055, -0.529171884059906, 0.9746371507644653, -0.47712934017181396, 0.05648805946111679, 0.7109766006469727, 0.6003644466400146, 0.1764293909072876, -1.5627591609954834, 0.6657454371452332, -0.9453895092010498, 0.8709739446640015, 0.3055022954940796, -2.0824460983276367, 0.5890249609947205, 0.5431345701217651, 0.38323837518692017, -2.8814337253570557, -1.191520094871521, -0.07363962382078171, 0.6198155283927917, 0.24588972330093384], [0.35182663798332214, 1.6304302215576172, 0.3876112103462219, 0.8969506621360779, 0.11129073798656464, -0.39144986867904663, -0.8220595717430115, -1.1097313165664673, -0.26954904198646545, -0.9160224199295044, 0.8922569155693054, 1.9969600439071655, -0.48203083872795105, 0.26533588767051697, 0.5529329180717468, -0.7128555178642273, 0.6526573896408081, 0.6651285886764526, 1.7318581342697144, 0.4038633704185486, 2.2355239391326904, 1.541543960571289, 0.3641965389251709, 0.5647661089897156, -2.5736050605773926, -0.7919505834579468, 0.28151196241378784, -0.4665062427520752, -0.6892073750495911, 0.3295910656452179, 1.089635968208313, -0.9926478862762451, -1.0085469484329224], [0.6275159120559692, 1.5346587896347046, -1.1334631443023682, 0.02817355841398239, 0.6252906322479248, 0.5334308743476868, -0.28574806451797485, 1.187665343284607, -2.1721975803375244, 0.5338695049285889, 1.3198728561401367, -1.7663013935089111, 0.6887795925140381, 0.27683329582214355, -0.39252912998199463, 0.7604058980941772, -2.947591543197632, 0.24232901632785797, -1.144324779510498, 0.496541291475296, -0.7322628498077393, -0.20198164880275726, -0.17420607805252075, -0.6220088005065918, -1.3383424282073975, -0.9606496691703796, 0.21336989104747772, 0.5068854093551636, 0.4573589265346527, 0.16988429427146912, 1.1507787704467773, 1.7800133228302002, -0.8515053391456604]], "dtype": "float32"}, 2, 16], "e": {"__tup__": [{"__nd__": [[2.5759315490722656, 8.496560096740723, -5.5024895668029785, -11.912447929382324, 5.424433708190918, -20.033681869506836, -15.703638076782227, 20.505870819091797, -8.11654281616211, -5.845582485198975, 21.47612953186035, -3.9446401596069336, 5.945959568023682, -0.7492080330848694, -5.521457672119141, 0.4290054142475128, -11.847057342529297, -13.754607200622559, -5.411484718322754, 7.111825466156006, -15.414093017578125, 3.01784348487854, -8.692113876342773, 17.52071762084961, -19.44657325744629, -5.7720184326171875, 14.474262237548828, 12.854921340942383, 11.967041015625, 0.8691312670707703, 17.354581832885742, -7.814894199371338, -9.505106925964355], [0.12051858007907867, 5.174094200134277, 2.6075196266174316, 1.352854609489441, -18.509872436523438, -13.423422813415527, 9.74535846710205, -8.6891450881958, -29.62203598022461, -5.147881031036377, 34.07223892211914, 13.765710830688477, 11.586641311645508, -8.795510292053223, 7.108414649963379, 14.903565406799316, -16.68556022644043, -23.319997787475586, -32.642215728759766, -13.43344497680664, -31.732614517211914, 5.255039215087891, 4.607137203216553, 1.7322957515716553, -9.031136512756348, 8.684172630310059, 14.070536613464355, 8.113293647766113, 23.189090728759766, 6.5583367347717285, -5.929430961608887, -3.6919729709625244, -9.974306106567383], [10.140698432922363, 20.894283294677734, 5.540857791900635, -3.6379055976867676, 14.345459938049316, 15.947355270385742, -18.360074996948242, -31.746278762817383, -12.452214241027832, -7.006719589233398, -34.39825439453125, -3.0241668224334717, -5.377763748168945, 7.695327281951904, -0.887913167476654, 12.58017349243164, -20.639652252197266, -1.734053134918213, 9.076042175292969, -0.08370408415794373, -6.56369686126709, 0.8752269148826599, 19.49354362487793, -1.8820923566818237, -3.8088464736938477, -15.07043743133545, 13.679878234863281, -26.03287124633789, -8.54926872253418, 1.2208001613616943, -6.3846235275268555, 5.604467391967773, 2.072782516479492], [8.499582290649414, 6.715397357940674, 7.261404991149902, 16.473033905029297, 12.10164737701416, -3.0055112838745117, 2.779323101043701, 0.5243290066719055, -5.370129585266113, 4.011817455291748, -7.6325788497924805, 4.660953521728516, -2.9611783027648926, 5.015618324279785, -19.375350952148438, 6.378015518188477, 13.554059982299805, 9.068325996398926, -6.468172550201416, 0.4598759412765503, -7.342014312744141, 11.196985244750977, -2.9169111251831055, -0.39991050958633423, -2.0937063694000244, 5.643995761871338, -6.2027812004089355, -21.880386352539062, 6.095950126647949, -8.691864013671875, -2.9195077419281006, -5.825263500213623, 4.46284294128418], [18.324710845947266, 12.958762168884277, -1.7183054685592651, 19.48592185974121, -26.294967651367188, 0.8706799149513245, -0.25987708568573, -1.005952000617981, 17.54436492919922, 1.1704957485198975, -0.3765089809894562, -25.128376007080078, 5.040135860443115, 12.183905601501465, -3.4502899646759033, -6.675971508026123, 4.692953109741211, -1.1075905561447144, -2.156195878982544, 17.86905860900879, 3.1006078720092773, -9.07358169555664, 10.35692310333252, 18.33883285522461, -13.192756652832031, 9.187573432922363, 1.1838561296463013, 33.23896026611328, -10.180886268615723, 6.90838623046875, 3.167780876159668, -5.4392194747924805, -2.962693929672241], [12.01082706451416, 16.053634643554688, 1.2376179695129395, -12.729626655578613, -13.196758270263672, -2.1418964862823486, 8.321614265441895, 0.2336510866880417, 4.523741722106934, 27.43757438659668, 0.1920791119337082, -22.515501022338867, 8.414911270141602, -15.318927764892578, 20.810319900512695, 3.44069242477417, 4.641827583312988, -6.202136039733887, -17.067907333374023, 1.404752492904663, -10.575874328613281, -11.690713882446289, -1.5744763612747192, 1.6598182916641235, -0.45330461859703064, -17.735183715820312, -0.5145164132118225, 30.86787986755371, 21.75899887084961, 18.915557861328125, -8.196245193481445, 13.508350372314453, 6.314602851867676], [-12.622883796691895, -0.3541809320449829, 6.336852550506592, 18.744489669799805, -0.1467568427324295, 6.517678737640381, -33.212371826171875, -4.116898536682129, 7.792393207550049, 5.5230255126953125, -5.009222507476807, 13.660821914672852, -3.9150397777557373, -7.377864360809326, -4.26991605758667, 6.990447521209717, -5.472963333129883, -9.271520614624023, -2.7505853176116943, 3.6326751708984375, -4.308269023895264, -4.027082443237305, 13.497904777526855, -3.7698211669921875, 5.836979866027832, -3.1371524333953857, -13.59228801727295, -1.4062893390655518, -8.508349418640137, -6.612856388092041, 2.8623037338256836, 2.956660270690918, 13.738712310791016], [-7.610380172729492, 5.313260555267334, -5.375377178192139, -16.765193939208984, 8.716060638427734, -6.25354528427124, 0.475195050239563, 6.461278438568115, 4.282382011413574, -1.9762675762176514, -7.189319133758545, -9.726505279541016, 12.872615814208984, 2.4191436767578125, 14.931790351867676, -6.670010566711426, 0.10489064455032349, 6.797775745391846, -6.550214767456055, 18.626710891723633, -17.25094985961914, 2.733583927154541, -13.514582633972168, -12.731084823608398, -9.343754768371582, 9.329939842224121, -2.610058307647705, -10.58471965789795, 0.9304419159889221, 14.55734920501709, -10.418242454528809, 17.08625602722168, 4.027620315551758], [-22.0393009185791, -0.12661510705947876, 5.073510646820068, 7.449777126312256, 2.2197532653808594, 9.504549026489258, -1.2459194660186768, 7.126643180847168, 7.009710788726807, -8.839751243591309, -4.028987884521484, -7.099264621734619, 17.293107986450195, 27.924745559692383, -2.192028760910034, 1.7535958290100098, 9.213376998901367, 6.857377529144287, -4.333523750305176, -12.996438980102539, 30.045520782470703, -7.241178035736084, -13.563464164733887, 8.176544189453125, -2.8564932346343994, -10.392024993896484, -26.789148330688477, -1.5331026315689087, 3.125439167022705, 9.077601432800293, -9.587484359741211, -10.33321475982666, -9.737695693969727], [-4.241837978363037, -1.7082245349884033, 14.422558784484863, -13.325438499450684, -0.19127574563026428, -5.316900253295898, 0.3531532883644104, -1.2929130792617798, 32.639495849609375, 3.257187604904175, -4.586037635803223, -3.429248571395874, -2.795685052871704, -12.019546508789062, -15.906611442565918, -9.194035530090332, 15.348601341247559, 6.463450908660889, -2.7602617740631104, -12.693359375, -4.61003303527832, -18.915861129760742, -7.15323543548584, 4.554893970489502, 11.642584800720215, 2.3756935596466064, 1.6405664682388306, 2.4137799739837646, -0.39329883456230164, -14.897578239440918, -10.420812606811523, 11.990468978881836, -11.118319511413574], [-7.2788238525390625, 18.956993103027344, -13.698387145996094, -7.856897354125977, -1.8749070167541504, -15.077760696411133, 10.605546951293945, -8.237710952758789, 4.812960147857666, 4.598195552825928, -2.5294759273529053, 16.6905574798584, 13.852314949035645, 11.089803695678711, -21.080469131469727, -0.8563233613967896, 31.00328826904297, 12.78719711303711, -6.891342639923096, 17.7485294342041, 20.94439125061035, -0.8041848540306091, 2.5568883419036865, -20.01727867126465, -13.731127738952637, 6.377348899841309, -5.9706926345825195, -8.675548553466797, -22.682207107543945, -4.802507400512695, 6.681019306182861, 10.239961624145508, -17.598594665527344], [-1.8800253868103027, -6.97622013092041, 18.036272048950195, 5.40150260925293, 21.37157440185547, 0.1349249929189682, -2.055068254470825, 1.417446494102478, 6.70041036605835, 8.762968063354492, -7.62327766418457, 0.947433590888977, 5.516175270080566, -12.708979606628418, -4.029420375823975, 16.060792922973633, -1.2970770597457886, 5.184540271759033, -5.6036295890808105, 18.84174156188965, -8.699771881103516, -8.006856918334961, -4.3223042488098145, 6.231215000152588, -1.221535563468933, 7.069404602050781, -9.374093055725098, 18.66929817199707, -4.51021671295166, 7.377519130706787, -0.1482752114534378, -3.9410417079925537, 15.384076118469238], [1.816525936126709, 8.653044700622559, -0.0408969409763813, -2.6045210361480713, 19.7810001373291, 18.701967239379883, -0.0732308030128479, 11.715754508972168, 6.177364826202393, 6.531977653503418, 10.114823341369629, 8.571076393127441, -21.34405517578125, 9.593255043029785, -6.232908248901367, 10.749603271484375, -10.201160430908203, -5.854750633239746, 5.102358818054199, 1.8283414840698242, -2.0801429748535156, 4.555368423461914, 20.30529022216797, 16.271425247192383, 20.663379669189453, -21.311689376831055, 7.375309467315674, 5.905975818634033, 6.355283260345459, -7.715969085693359, 20.38151741027832, 1.2389426231384277, -3.1415727138519287], [0.4681861400604248, 16.74176025390625, -2.847987413406372, -18.508563995361328, 12.085409164428711, 11.085847854614258, 18.129459381103516, -14.204973220825195, -5.2532958984375, 20.29641342163086, 15.556211471557617, -8.687664031982422, -12.345135688781738, -33.753597259521484, 10.913498878479004, 3.068065881729126, -13.760649681091309, -8.539344787597656, 8.872832298278809, 0.9229542016983032, 22.609128952026367, 6.712807655334473, 2.1766250133514404, -4.264909267425537, -4.4359846115112305, -14.681262016296387, 7.718846797943115, -13.737015724182129, 4.334819316864014, 10.228835105895996, -2.0108284950256348, 22.517744064331055, 7.346557140350342], [17.40658950805664, 15.583683013916016, -3.530881881713867, 12.266595840454102, -9.639945030212402, 3.863725423812866, -1.1899086236953735, 4.6657257080078125, -14.72821044921875, 5.230085849761963, 15.202856063842773, -33.42402267456055, 8.265033721923828, -4.577596187591553, -2.668689727783203, 13.347952842712402, 1.3975272178649902, -8.753355026245117, -19.328876495361328, 14.20896053314209, -16.805652618408203, -0.055669885128736496, -5.360625743865967, -0.3669601380825043, 0.5797024369239807, -6.204467296600342, -0.03682582452893257, -0.344195693731308, 23.61727523803711, -4.351981163024902, -8.874319076538086, -19.834739685058594, 1.404732346534729], [-6.372551918029785, 20.48832893371582, 14.153026580810547, 2.5928704738616943, 9.3831205368042, 3.455667018890381, 7.724227428436279, 3.134037494659424, -7.0125603675842285, 4.417718410491943, -7.884580135345459, 9.263157844543457, -5.50392484664917, -12.921080589294434, 15.892264366149902, -3.878011703491211, 15.147473335266113, 1.5951937437057495, -11.286836624145508, -1.2772387266159058, -1.609020709991455, 12.71677303314209, -11.861096382141113, -1.7631162405014038, 7.219735145568848, -2.216055154800415, 17.947439193725586, 18.041677474975586, -8.548376083374023, -9.064016342163086, -19.161365509033203, -17.611061096191406, -12.875160217285156], [-2.4918386936187744, 18.357845306396484, 0.795417308807373, -4.3255720138549805, -1.8913809061050415, 12.781146049499512, -5.784887790679932, -4.8993682861328125, -5.245645523071289, 17.49964714050293, 3.9258980751037598, 8.655203819274902, -11.978821754455566, -1.3564497232437134, -4.2834391593933105, 8.669384956359863, 24.355026245117188, -24.440765380859375, 3.784691572189331, -0.7768046259880066, -1.9733995199203491, 15.356524467468262, -5.04032564163208, -9.595508575439453, 9.18166446685791, 0.26809048652648926, 9.862919807434082, -8.766240119934082, 11.998878479003906, -5.154107093811035, -4.318909168243408, 14.961819648742676, 4.081546306610107], [-10.28320026397705, -20.60307502746582, 4.738803386688232, -15.428050994873047, -7.0626068115234375, -5.99688720703125, -12.30322551727295, -12.215423583984375, 10.203252792358398, -7.954587936401367, -19.4368896484375, 10.714735984802246, 9.409146308898926, -3.3445866107940674, 7.906515598297119, -3.045957565307617, 27.223684310913086, 2.586296796798706, -15.679072380065918, -16.77033233642578, -10.39303970336914, -2.1126716136932373, -9.822955131530762, 4.8612470626831055, 19.359272003173828, 6.641472816467285, 5.639610767364502, 4.069706916809082, 4.804420471191406, 3.675065755844116, -7.254749774932861, -4.302969455718994, 5.354995250701904], [0.45740875601768494, -1.7609941959381104, -4.222957611083984, 5.937121391296387, -9.21086597442627, 1.0037258863449097, -11.557847023010254, -14.373993873596191, 9.059364318847656, 11.13963508605957, -1.968280553817749, 11.53825855255127, -7.735063076019287, 2.389131784439087, 1.5978362560272217, -7.710153102874756, -7.464211940765381, 10.773175239562988, -4.412874221801758, 4.09087610244751, 2.698476552963257, 9.915164947509766, 3.804475784301758, -0.9871727228164673, 6.912741184234619, 5.470186710357666, -4.800184726715088, 6.2430620193481445, 3.513252019882202, -5.783233642578125, 16.776832580566406, 16.374832153320312, -1.0911078453063965], [2.0990817546844482, 24.47736167907715, 5.243889331817627, 4.354956150054932, 5.936878681182861, 7.061558246612549, -2.3154897689819336, 27.32271957397461, -3.9345922470092773, -1.1052274703979492, -32.905662536621094, 5.3838701248168945, 10.199895858764648, 14.048564910888672, -25.77237892150879, 14.037575721740723, -9.74630355834961, -0.8082107901573181, -8.28794002532959, 2.0747804641723633, -20.463533401489258, -1.2072266340255737, 2.091696262359619, 8.995424270629883, 8.956214904785156, -4.901674747467041, -4.817831039428711, -5.8988189697265625, 5.541402339935303, -4.052184581756592, -9.526789665222168, -25.53234100341797, 9.198196411132812], [3.696394920349121, 1.7940025329589844, 6.12214994430542, 4.7317423820495605, 10.500128746032715, 18.579967498779297, -3.453385591506958, 13.704174041748047, -16.524263381958008, 8.070602416992188, 8.791522979736328, -0.5186423659324646, 6.546914577484131, 6.95187520980835, 3.9153690338134766, -20.10618019104004, -1.970205307006836, 8.956005096435547, -3.7735824584960938, -6.932105541229248, -4.8106889724731445, 16.737070083618164, 7.86725378036499, 1.9272500276565552, 10.19211483001709, -12.854053497314453, -12.565653800964355, -6.120206832885742, 22.784833908081055, 13.808152198791504, -39.58879089355469, 6.900503635406494, -12.929363250732422], [-16.975749969482422, 17.322153091430664, 1.8475273847579956, -15.923751831054688, 0.2122086137533188, 14.79758358001709, -13.4840087890625, 14.691479682922363, 12.177779197692871, 14.243112564086914, -1.1530916690826416, 5.97263765335083, -2.119385004043579, -17.174606323242188, -0.8873248100280762, -1.1839954853057861, 0.8819775581359863, -20.132652282714844, 6.532881736755371, -16.949684143066406, 6.42357063293457, -8.230173110961914, -3.229092836380005, -14.487574577331543, -7.412411212921143, -3.517641544342041, -1.303741455078125, 5.859881401062012, -2.5406084060668945, -1.3851337432861328, 9.287151336669922, -19.78319549560547, 0.20296669006347656], [7.234312057495117, 10.006433486938477, -4.81148099899292, 12.065529823303223, -6.631796360015869, -6.463132858276367, 9.391435623168945, 5.517712116241455, 2.577100992202759, 7.193902492523193, -4.071719169616699, 12.002254486083984, 0.13294968008995056, 23.263004302978516, 21.264060974121094, -9.505088806152344, 5.0780816078186035, 41.44427490234375, 7.970993995666504, 3.4607975482940674, 19.576383590698242, 2.135660409927368, 13.467576026916504, 2.724545478820801, -10.92223072052002, -6.863986492156982, -20.96000099182129, -1.9720345735549927, 3.4445302486419678, 14.069500923156738, -8.522246360778809, 3.4620327949523926, -7.160105228424072], [-5.696027755737305, -2.751985788345337, 21.680753707885742, 5.211338996887207, 23.838136672973633, 0.6444129943847656, 4.567944049835205, -6.408308506011963, -5.5430426597595215, 4.627617359161377, 27.107938766479492, 0.19022934138774872, 1.1693813800811768, -11.480582237243652, 1.2674678564071655, -11.150150299072266, 13.643675804138184, -7.335233688354492, 6.4926347732543945, -10.740314483642578, -8.938302993774414, 1.8235507011413574, 4.575414657592773, -1.9358185529708862, -12.738495826721191, 12.202411651611328, 9.59953498840332, -25.755794525146484, 6.0254902839660645, 18.079914093017578, -8.190653800964355, -17.063472747802734, -19.63886070251465], [3.830566644668579, 9.93834114074707, 10.30652904510498, -0.617732584476471, -2.4755077362060547, 11.599050521850586, 5.948479652404785, 11.578969955444336, -19.492643356323242, 3.1531789302825928, 9.077701568603516, -4.468324184417725, -6.311668395996094, 12.55958366394043, -12.449678421020508, -2.1307523250579834, -13.360871315002441, -13.545767784118652, -2.3411436080932617, -30.48370933532715, 0.9193195700645447, 2.5596632957458496, 10.626535415649414, 1.2437713146209717, -4.173793315887451, 13.11150074005127, 0.8465081453323364, -8.288971900939941, -2.29980206489563, -14.174261093139648, 2.1557457447052, 2.6398701667785645, -3.4857892990112305], [-9.210917472839355, 1.2587072849273682, -10.2263822555542, 4.118289947509766, 15.044719696044922, 16.893701553344727, 17.681903839111328, 10.74599838256836, -8.91730785369873, -4.165534496307373, 11.402935981750488, -2.6095943450927734, 19.52479362487793, -4.768921375274658, 11.129541397094727, 15.540562629699707, 3.121760845184326, 2.459817409515381, -21.30958366394043, -9.225214958190918, -4.9879679679870605, -17.850543975830078, 10.198783874511719, -9.829411506652832, -7.315019130706787, 2.758944034576416, -10.105313301086426, -2.486537218093872, 16.362619400024414, 0.2191569209098816, 16.977243423461914, 1.5674563646316528, 11.670485496520996], [5.726241111755371, -9.3955717086792, -8.790313720703125, 0.13409213721752167, -1.392993450164795, 7.636810302734375, -7.716928005218506, -23.801015853881836, 14.328587532043457, -16.73004150390625, -10.100640296936035, 6.9110541343688965, 13.394350051879883, 13.177348136901855, 14.459222793579102, -1.448594331741333, 5.536480903625488, -10.450557708740234, 4.686399936676025, -19.545490264892578, -10.38776969909668, 0.25125864148139954, 10.878654479980469, -5.03502082824707, -8.816826820373535, -17.331493377685547, 6.884206771850586, -8.71203899383545, 24.857553482055664, 11.709463119506836, -4.810138702392578, 1.2722749710083008, -8.292839050292969], [-8.783435821533203, 11.359460830688477, 1.205466628074646, -9.131952285766602, -17.761009216308594, -6.368170738220215, -8.80054759979248, -13.351655006408691, 13.442597389221191, 3.8457412719726562, -14.373727798461914, 9.774104118347168, 7.1418867111206055, -1.8576157093048096, -2.463972330093384, 4.744541168212891, 0.15603946149349213, 1.314686894416809, 12.780656814575195, 11.368550300598145, 0.04019935056567192, -17.23240089416504, -8.43375015258789, 3.22843337059021, 4.143476486206055, 2.766648292541504, -14.740514755249023, 12.330269813537598, -22.47345542907715, -5.164340496063232, 8.478538513183594, 4.471088409423828, 31.29937171936035], [-7.571414470672607, -23.926034927368164, -12.479695320129395, -17.09494972229004, -4.869223594665527, -9.091954231262207, -19.678369522094727, -0.6819857954978943, -6.692303657531738, -21.427682876586914, -0.5768039226531982, 18.894872665405273, -4.147971153259277, -10.810760498046875, 11.534201622009277, 4.969138145446777, 13.296943664550781, -20.513019561767578, -18.084625244140625, 9.015151023864746, 3.7317233085632324, 3.7737741470336914, 1.9596234560012817, -4.825448513031006, 0.7901990413665771, -1.9585151672363281, 0.522936224937439, -26.45734977722168, -1.7202595472335815, -3.880955934524536, -11.692267417907715, -3.372584581375122, 2.249042272567749], [-2.998089551925659, -21.96689796447754, 9.078063011169434, -0.30362415313720703, -7.926709175109863, -10.584296226501465, 5.562345504760742, 4.756414413452148, 11.631080627441406, 6.191421031951904, -2.730077028274536, -14.197705268859863, -10.258872032165527, 18.035022735595703, -0.8643723726272583, -5.641389846801758, 14.895252227783203, -1.4585840702056885, -13.161219596862793, 1.0837775468826294, -20.60211181640625, -10.203149795532227, 5.854065895080566, 18.84015655517578, 0.288846492767334, 2.219125747680664, -11.083259582519531, -9.46672534942627, -8.72680950164795, 2.0166189670562744, 2.5056259632110596, 3.571563720703125, 15.96467399597168], [9.21619701385498, 12.635313034057617, -5.665089130401611, 6.632288455963135, 0.12419730424880981, 11.96521282196045, -28.629253387451172, 17.962121963500977, 1.1924391984939575, -10.537602424621582, -4.2615580558776855, -15.336301803588867, 5.736416339874268, 7.674252510070801, -0.7225729823112488, 21.07748794555664, -32.604183197021484, 9.013636589050293, -10.994169235229492, -15.525535583496094, 16.822965621948242, -2.7295851707458496, 10.224559783935547, -1.3834270238876343, 9.497373580932617, -11.908936500549316, -25.751182556152344, 2.623011350631714, 12.695588111877441, -10.363668441772461, -14.590665817260742, 9.6402006149292, -7.723499298095703], [-9.314136505126953, -19.65911865234375, -1.4291950464248657, 4.308740139007568, 1.1978843212127686, -8.756278038024902, 6.901153564453125, 3.058948516845703, -6.313475131988525, 15.98080062866211, -14.400544166564941, -1.8184541463851929, 17.664474487304688, -9.710896492004395, 6.2976460456848145, -12.328550338745117, -1.6302368640899658, 11.23606014251709, -6.710940361022949, 19.114301681518555, 18.132266998291016, -13.483866691589355, -41.213531494140625, 16.345504760742188, 8.224082946777344, 1.4737039804458618, -22.634809494018555, 14.921647071838379, -10.998432159423828, -4.357450008392334, 2.698711633682251, 18.761215209960938, 24.3515567779541], [2.4161949157714844, -7.770349502563477, 15.804877281188965, -8.974003791809082, -0.7642809748649597, -9.528464317321777, 16.300344467163086, -17.258573532104492, -0.5294091701507568, -10.461730003356934, 2.1118128299713135, -16.80844497680664, -10.490180969238281, -25.697668075561523, -9.31544303894043, -9.27934741973877, 5.8773932456970215, -16.274248123168945, -24.151519775390625, -9.670324325561523, 3.0557405948638916, -21.10785675048828, 13.548901557922363, -9.119281768798828, -12.218737602233887, 4.153172016143799, -0.02000347338616848, -3.8713295459747314, -2.3491389751434326, -30.709121704101562, 5.995354652404785, 4.6337151527404785, -23.04119873046875], [23.87932777404785, -6.823657512664795, 12.327757835388184, 12.75806999206543, 19.466733932495117, 3.624159812927246, 8.533126831054688, -6.6522746086120605, 3.167562484741211, -10.472640991210938, -5.130210876464844, 4.264257431030273, 21.269437789916992, -4.659067630767822, -3.200756072998047, -7.679771900177002, 4.142701625823975, 25.282480239868164, 4.263058185577393, 2.4887168407440186, 10.619611740112305, 8.319663047790527, -13.359091758728027, -7.650371074676514, 6.381194114685059, 20.46971893310547, -27.024049758911133, -1.2765840291976929, -11.557137489318848, 10.527999877929688, -11.353605270385742, -7.602372169494629, -4.756407260894775], [-10.866902351379395, -16.6199951171875, 4.338414669036865, 0.8664493560791016, -12.989901542663574, -5.967987537384033, 11.228604316711426, 15.667072296142578, 14.639790534973145, -3.087278366088867, 9.603214263916016, 12.12496566772461, -22.82622718811035, 12.550052642822266, -9.963001251220703, -11.439445495605469, 25.58740997314453, 5.303391456604004, -13.880242347717285, -0.7634370923042297, 16.53380012512207, -2.9288461208343506, 7.366787910461426, -4.779480934143066, -9.1319580078125, 23.61847686767578, 14.291670799255371, 3.836548089981079, -13.514116287231445, -9.744938850402832, -14.247444152832031, 15.857466697692871, -21.45421028137207], [10.214649200439453, 8.459013938903809, -21.919057846069336, -0.20591618120670319, 3.1621429920196533, -5.025450706481934, 10.098560333251953, 12.193133354187012, -3.6870667934417725, -0.17513199150562286, 18.257505416870117, 9.46599292755127, -5.9314727783203125, 2.887578010559082, 8.585480690002441, -3.731714963912964, 8.997880935668945, -2.0922610759735107, 0.8859129548072815, 13.985786437988281, -0.34887975454330444, 7.068955421447754, -22.139890670776367, 7.112814426422119, -2.178990602493286, -2.9037575721740723, -1.776961088180542, 6.7756757736206055, -10.669024467468262, -0.40303200483322144, 28.27085304260254, 21.804048538208008, 4.797661304473877], [-6.430335521697998, 2.007356882095337, 1.3338667154312134, -0.35706350207328796, -6.968592643737793, -26.16437530517578, -5.018433094024658, -2.6136763095855713, 11.830679893493652, -22.433481216430664, -33.15150451660156, 8.679840087890625, -19.427024841308594, 3.1441001892089844, -5.229389190673828, 6.984947204589844, 22.306921005249023, -1.012494683265686, -10.50869083404541, 4.681032657623291, 16.906909942626953, -2.9903054237365723, -18.878633499145508, -6.301401138305664, 18.04043197631836, -8.639802932739258, -4.4820556640625, -5.073726177215576, -25.273664474487305, 17.97298812866211, 12.590475082397461, 9.814587593078613, 8.388988494873047], [-13.923123359680176, 11.387530326843262, 9.939484596252441, 3.7764928340911865, 17.777875900268555, 19.16215705871582, 4.105603218078613, -1.6570055484771729, 6.0150346755981445, 25.98577117919922, -5.305443286895752, 8.141704559326172, -8.114581108093262, 4.734614849090576, -15.204450607299805, -6.998385429382324, -10.367656707763672, -0.22776421904563904, 0.27416232228279114, -13.566149711608887, 10.045331954956055, 14.104125022888184, 25.44644546508789, 13.792121887207031, -20.44808006286621, -13.341155052185059, -4.266611576080322, -2.5358545780181885, 14.024559020996094, -13.205727577209473, -12.762048721313477, -4.070244789123535, -22.927385330200195], [9.068790435791016, -3.709791660308838, 2.245912551879883, 7.303682804107666, 2.098137140274048, 9.482707023620605, -8.560721397399902, 5.780862331390381, 22.687217712402344, 0.883856475353241, 12.164536476135254, 14.97892951965332, -6.607480049133301, 8.65944766998291, 8.326363563537598, 0.46377572417259216, 4.077240467071533, -10.073404312133789, 11.976880073547363, -3.839001178741455, 2.3356897830963135, 0.3409816026687622, 14.102437019348145, -23.432437896728516, 6.613450527191162, 0.38347652554512024, -11.193981170654297, -14.270794868469238, -1.7555090188980103, -16.531536102294922, 10.975756645202637, -11.47200870513916, -2.9024507999420166], [-5.361754894256592, -11.95612907409668, 1.5153981447219849, 7.670370101928711, 11.710693359375, 32.06434631347656, -3.7161977291107178, 1.0513502359390259, -15.519013404846191, -17.779422760009766, -13.658936500549316, 15.634659767150879, 3.346000909805298, 6.444675922393799, -2.9495017528533936, 0.902925431728363, -12.998340606689453, -0.14322467148303986, 3.7996370792388916, -0.642874002456665, 6.1981587409973145, -1.8393371105194092, -16.94570541381836, -6.940005302429199, 19.31468391418457, -11.625693321228027, -14.417353630065918, -8.78617000579834, 4.72954797744751, -8.523929595947266, 11.102189064025879, 5.088204383850098, 2.62654447555542], [1.387562870979309, 1.7940000295639038, 4.5327630043029785, -1.4105844497680664, 3.9879605770111084, 14.144658088684082, -10.640508651733398, -2.117577314376831, 7.507514476776123, -21.769168853759766, -5.152176380157471, 19.505693435668945, 7.760059833526611, 2.286857843399048, -3.3212671279907227, 3.175841808319092, -8.770813941955566, -28.136001586914062, -12.418394088745117, -15.572911262512207, 11.528666496276855, -4.1574296951293945, 22.547767639160156, -7.638561248779297, -9.447407722473145, 1.5449739694595337, 14.80881404876709, -11.129215240478516, 0.646009087562561, -15.95980453491211, 14.477991104125977, -11.17155647277832, -1.805842399597168], [-0.6696742177009583, -2.696056604385376, -0.8208519816398621, 10.899145126342773, -9.943864822387695, -19.28485107421875, -7.260435581207275, 3.2176308631896973, -8.67298412322998, -0.46907496452331543, 3.654294013977051, -13.969693183898926, -4.768168926239014, -6.946017265319824, 19.195722579956055, 0.6319079995155334, -6.097105979919434, 3.3821043968200684, -2.8692467212677, 1.1220077276229858, -14.987557411193848, -16.616775512695312, -3.6085033416748047, 8.244224548339844, 9.239128112792969, 21.616708755493164, -13.261571884155273, -0.34897518157958984, -2.77428936958313, 5.4307074546813965, -16.699758529663086, 8.060586929321289, 6.005686283111572], [-0.1952793002128601, 10.735736846923828, 10.426958084106445, -20.51302719116211, 10.741915702819824, 9.795260429382324, 1.8883057832717896, 15.910585403442383, 20.796541213989258, 3.573249340057373, 0.9415309429168701, -21.883106231689453, 9.971854209899902, -14.840065002441406, -22.83324432373047, -6.636280536651611, -25.686641693115234, 9.27121353149414, 1.547205924987793, 4.2551398277282715, 5.8758063316345215, -6.898808002471924, -15.249700546264648, -1.166927456855774, 7.340555667877197, -14.52517032623291, -1.3747174739837646, 28.637475967407227, 17.65923500061035, -7.532351016998291, -3.0687148571014404, 8.980269432067871, 14.232349395751953], [7.36997127532959, 6.499485492706299, -13.205514907836914, -2.7630772590637207, -8.054387092590332, 12.03929328918457, 2.671966791152954, 2.120000123977661, -26.62181854248047, -3.9465672969818115, 11.991971969604492, 0.9603410363197327, 6.82797384262085, 2.782388210296631, 3.7335569858551025, -17.603654861450195, -13.220540046691895, 22.716697692871094, 17.256959915161133, 5.497138500213623, 8.349443435668945, 14.911614418029785, -8.732972145080566, -20.040088653564453, 1.687036395072937, 16.07066535949707, -5.469717025756836, -1.459523320198059, -4.6416707038879395, -5.520549297332764, -6.4245429039001465, -4.372176170349121, -19.694482803344727], [3.78617000579834, -9.894122123718262, -2.4481327533721924, -5.759958744049072, -14.4102783203125, -17.238231658935547, 12.418065071105957, -12.842803001403809, 0.81681227684021, 2.2587690353393555, 10.435920715332031, 7.730078220367432, 11.763205528259277, 13.869820594787598, -6.494078159332275, 8.000797271728516, 2.2466554641723633, -7.777679443359375, -18.027530670166016, -9.972031593322754, -6.465799808502197, 1.5814529657363892, 0.3189307153224945, -3.8844971656799316, -3.0187108516693115, -12.526183128356934, 16.52567481994629, 11.942192077636719, 12.808736801147461, 1.747652530670166, -2.829376220703125, 5.951098918914795, 6.841536045074463], [12.426558494567871, -12.161319732666016, -7.1557159423828125, -9.112958908081055, -6.809795379638672, -4.769225120544434, 0.12900471687316895, -10.570788383483887, 15.847453117370605, -0.4144933521747589, -7.290045738220215, 11.113024711608887, -4.199859619140625, -10.028942108154297, 3.552448034286499, -2.2838542461395264, -0.41304630041122437, -10.187137603759766, 12.738158226013184, 11.148687362670898, -25.74604606628418, 1.4036831855773926, -1.532151222229004, -13.605854034423828, -2.0452351570129395, 3.4745800495147705, 17.211320877075195, 16.86005401611328, -9.804431915283203, -18.96030616760254, -11.870068550109863, 16.082069396972656, 3.9803695678710938], [16.096248626708984, -18.998321533203125, 3.203097343444824, 3.3845736980438232, 5.399630069732666, -10.136181831359863, -12.815159797668457, 13.37345027923584, 6.456109046936035, -11.024909019470215, -10.25098991394043, -14.707298278808594, -2.993927478790283, -5.623348236083984, -14.330021858215332, -6.363287448883057, -1.777238368988037, 0.2888447046279907, -4.781391620635986, 32.23694610595703, 0.4640267491340637, -22.989383697509766, -11.05482292175293, 3.1803414821624756, 17.805160522460938, 2.7783234119415283, 7.268111705780029, -5.612593173980713, -19.465036392211914, 1.6788382530212402, -10.028719902038574, -10.024019241333008, 12.698018074035645], [-15.884211540222168, -4.29110860824585, -7.39922571182251, -7.415290832519531, -2.3326284885406494, 16.44318199157715, 12.064022064208984, 12.944643020629883, -24.267850875854492, 7.508569240570068, -4.988691329956055, -0.3480179011821747, -13.773500442504883, 11.704102516174316, -0.48703253269195557, 9.85800838470459, -9.206988334655762, -13.642675399780273, -27.236331939697266, 6.3452677726745605, 15.555957794189453, 17.082271575927734, 16.433303833007812, 6.324695110321045, -18.242691040039062, -0.27901917695999146, 17.94428253173828, -14.132136344909668, 25.071138381958008, -10.052289009094238, -21.56658172607422, 26.955177307128906, -5.928096294403076], [25.777616500854492, 31.661666870117188, 14.201766967773438, -28.621566772460938, -0.17707721889019012, -4.527101993560791, 3.2525875568389893, 2.7465784549713135, 2.4260690212249756, 21.06195831298828, -19.998764038085938, 11.59483814239502, 4.520010471343994, -19.36668586730957, -18.261699676513672, 9.434642791748047, 6.962098121643066, 21.393356323242188, 3.7883551120758057, -3.5222671031951904, 11.507796287536621, -3.7612695693969727, 16.14251708984375, 10.414780616760254, -8.374492645263672, -11.292363166809082, -3.5315604209899902, 23.824617385864258, -19.12433433532715, 14.06095027923584, -1.8379039764404297, 4.456112384796143, 3.2170779705047607], [-4.930227756500244, -3.2172305583953857, -8.369906425476074, 7.310582637786865, -5.29913330078125, 15.289084434509277, -1.1694436073303223, -3.717902183532715, 5.378284931182861, 9.749267578125, 10.094444274902344, -11.434652328491211, -4.465129375457764, -15.507445335388184, 0.16695451736450195, -10.43272590637207, 1.2303909063339233, 5.965651512145996, 1.4711436033248901, -0.702703058719635, 1.5351113080978394, -8.786211013793945, 3.9566938877105713, -12.942378997802734, -7.81922721862793, 6.819950580596924, 9.090176582336426, -4.196259021759033, -7.3950395584106445, -11.445348739624023, -0.8458298444747925, -19.43240737915039, -0.9715638756752014], [-0.15013615787029266, 7.457366943359375, -13.531036376953125, -9.119973182678223, 7.9073166847229, 2.8392367362976074, -2.068201780319214, -2.7623603343963623, -18.492170333862305, -24.399948120117188, -25.171470642089844, 10.707860946655273, -18.66571044921875, 20.923561096191406, -4.350851535797119, 13.480281829833984, -5.252369403839111, 2.0415611267089844, -10.252923965454102, -2.723677396774292, -10.2123384475708, -0.10456530004739761, 15.651480674743652, -2.8524997234344482, 21.225236892700195, -1.4587920904159546, -3.4893057346343994, -15.845054626464844, -3.0934200286865234, 2.6700353622436523, -3.3504550457000732, -2.113757371902466, -0.4199523329734802], [-12.108026504516602, -14.292174339294434, -12.576323509216309, -6.92664909362793, 24.182565689086914, -6.191699028015137, 1.9267990589141846, -8.936580657958984, -13.025683403015137, -9.968759536743164, 8.266562461853027, 10.314640045166016, 4.964743137359619, -9.033848762512207, 0.16696412861347198, 13.711241722106934, 7.307781219482422, 0.04268950596451759, 15.287481307983398, 3.152587413787842, 9.097414016723633, -8.738614082336426, 3.1134350299835205, -20.754770278930664, -18.594375610351562, -16.613229751586914, 12.521489143371582, -21.81351089477539, -17.3494815826416, -1.4770616292953491, 37.853546142578125, 19.980690002441406, -10.488130569458008], [-6.679008960723877, 2.157796621322632, 9.816061973571777, 16.158435821533203, 7.253591060638428, 15.312491416931152, -9.696939468383789, 11.249161720275879, 1.6202704906463623, 13.219160079956055, 8.262441635131836, -4.277827739715576, -1.4461034536361694, -2.2117929458618164, -9.245451927185059, -2.807560920715332, -0.5026925802230835, -14.07906723022461, -18.240949630737305, -11.264899253845215, -11.36178207397461, -1.7017412185668945, -12.751765251159668, 0.5649585127830505, 6.427768230438232, 11.541450500488281, 1.391265869140625, 12.49660587310791, -2.876032590866089, 10.461133003234863, -10.734503746032715, 4.394406795501709, 7.310108661651611], [-2.389373302459717, 31.834278106689453, 3.076226234436035, 3.8885722160339355, 4.358806133270264, 10.88579273223877, -11.447839736938477, 1.3341622352600098, -6.330865859985352, 9.29173469543457, 10.56202220916748, -26.87432289123535, -1.9886324405670166, 2.0387377738952637, -6.844427585601807, 8.082582473754883, -1.307051181793213, 14.914566040039062, 3.61549973487854, 3.8075945377349854, 10.48581314086914, 7.469485759735107, 2.6967005729675293, 4.284167766571045, -1.543599247932434, 8.737357139587402, -8.799753189086914, 6.052525043487549, 14.965609550476074, 4.424316883087158, 13.094376564025879, -7.7278733253479, -19.222246170043945], [-1.8929601907730103, 19.32891273498535, 1.1723644733428955, -5.452123641967773, -9.390323638916016, 3.9282586574554443, 10.465216636657715, 4.842136859893799, -9.846537590026855, -2.044081449508667, -11.998945236206055, 2.1608102321624756, -13.627591133117676, -12.785483360290527, -9.882024765014648, 13.016058921813965, 7.581071853637695, -11.318243980407715, 16.639801025390625, -1.7083444595336914, 11.038240432739258, 5.6546831130981445, 0.883672833442688, 12.492232322692871, -0.8690023422241211, -1.03732430934906, -7.177881717681885, 2.199962615966797, 10.07690715789795, -7.064858436584473, -15.83662223815918, -10.398340225219727, -19.217235565185547], [-5.964837074279785, 0.9235948324203491, -6.686578750610352, -12.6255521774292, -7.716685771942139, -5.782786846160889, -5.986137866973877, 2.5227413177490234, 10.326444625854492, -8.832387924194336, -43.35843276977539, -12.038034439086914, -4.110759735107422, 14.829814910888672, -7.027561187744141, -13.338626861572266, 13.332606315612793, -15.333747863769531, 8.121513366699219, -0.8900948762893677, -1.4453884363174438, -7.26515531539917, 17.770586013793945, -19.34473419189453, -1.4730950593948364, 14.74929141998291, 13.614006042480469, -6.156320095062256, -16.041170120239258, -24.274612426757812, 18.588029861450195, -7.566433429718018, -17.819875717163086], [0.964790940284729, 4.294834136962891, 17.649234771728516, -3.5394046306610107, -3.6676058769226074, 9.49254035949707, 17.73561668395996, -4.003355503082275, 16.717952728271484, -0.22247397899627686, -0.00925889890640974, 19.50760269165039, 18.340166091918945, -9.864563941955566, 9.851714134216309, -0.42584168910980225, 10.380853652954102, -11.927566528320312, -2.6662492752075195, -4.509288787841797, -4.255688190460205, 3.0695266723632812, 1.412764549255371, 12.279988288879395, -0.7406099438667297, -6.880224704742432, 2.404057502746582, 1.6970502138137817, 8.62475872039795, -5.6569013595581055, -15.271050453186035, 17.839250564575195, -13.027649879455566], [9.166903495788574, -10.296252250671387, -6.490565776824951, 3.0071682929992676, 6.47379732131958, 18.192951202392578, -6.105521202087402, -17.205320358276367, 11.608521461486816, -7.080589294433594, 9.228530883789062, -1.012905478477478, 0.25290292501449585, 12.155856132507324, -4.374603271484375, -13.273062705993652, -8.206236839294434, 17.594345092773438, -3.5741701126098633, 7.018101692199707, 9.561614990234375, 3.613154172897339, 12.693034172058105, -15.780834197998047, 14.118487358093262, -21.624887466430664, 2.1146128177642822, 2.016714334487915, -5.729559898376465, -18.33340072631836, 7.591493129730225, 12.71668529510498, -0.7196637392044067], [0.8399118781089783, -16.08639144897461, 17.665241241455078, 6.2156805992126465, 16.47857093811035, -0.8134753704071045, 10.79226016998291, -6.223507404327393, 7.120569229125977, 14.957295417785645, 14.042095184326172, 0.29518723487854004, 1.615000605583191, -19.087663650512695, 9.997474670410156, -1.615578532218933, -1.9395008087158203, 8.620586395263672, -16.280723571777344, 14.74936294555664, -14.670891761779785, 2.5864109992980957, -15.606971740722656, -0.3099832236766815, -22.21117401123047, -9.279282569885254, 6.3458123207092285, 19.132768630981445, 3.7233355045318604, 7.7087626457214355, -0.5293172597885132, 10.6136474609375, -7.364672660827637], [4.0553669929504395, -1.0502558946609497, 0.7685403227806091, -2.7003681659698486, 16.029769897460938, 9.05307388305664, -10.328164100646973, 8.178553581237793, 8.246975898742676, 9.743389129638672, 7.268466949462891, -0.15261204540729523, 1.5412259101867676, -12.08945083618164, -11.775930404663086, 4.165945529937744, 16.171974182128906, -3.3494648933410645, -0.9214537739753723, -4.021816730499268, -13.531685829162598, -2.620131731033325, -8.740476608276367, -12.955368995666504, 14.303828239440918, 7.751135349273682, 19.351869583129883, -6.561463832855225, 2.2139172554016113, 9.308073043823242, 3.4533262252807617, 2.3953752517700195, 10.571167945861816], [5.123950481414795, 18.496591567993164, 17.96619987487793, -8.206104278564453, 0.36448100209236145, 9.625130653381348, -13.225302696228027, 19.045486450195312, 6.258360862731934, 16.77351951599121, -15.571261405944824, 18.25480842590332, -21.551450729370117, -0.5363707542419434, -13.899673461914062, 3.7501730918884277, -14.798394203186035, 19.345680236816406, -3.178386688232422, 24.31451988220215, -5.991186141967773, 22.161386489868164, 11.223295211791992, 13.985169410705566, -0.25045397877693176, -8.97282600402832, 12.094242095947266, -0.5970789790153503, -2.0312914848327637, -4.42064094543457, -17.25855255126953, -1.7928614616394043, 3.312570571899414], [0.20717336237430573, 5.216858386993408, 8.011000633239746, 5.580502510070801, 10.423579216003418, -12.55133056640625, 15.610995292663574, 18.102996826171875, -0.27578985691070557, -8.06694507598877, -10.330853462219238, 7.964433670043945, 6.930881977081299, -2.231832265853882, -19.160409927368164, -10.0004243850708, -10.82688045501709, -4.351833820343018, 10.286847114562988, 0.3191278278827667, 11.424636840820312, -0.6127399206161499, -1.6361619234085083, 13.650788307189941, -9.267251014709473, -6.52593994140625, 1.7972698211669922, 24.896753311157227, -9.913167953491211, 8.195150375366211, 6.347612380981445, 9.754615783691406, 4.9354658126831055], [10.477495193481445, -17.780797958374023, 3.0685179233551025, 17.861244201660156, 10.308126449584961, 8.105338096618652, 7.962169647216797, -3.5142977237701416, 1.2292240858078003, 6.121362686157227, 16.44782829284668, -24.022798538208008, 21.375864028930664, -30.58348274230957, 8.641517639160156, -9.176252365112305, -6.613866806030273, 23.230886459350586, -3.370074987411499, 2.296128273010254, 18.01789093017578, -2.921222686767578, -24.680679321289062, -11.133133888244629, -21.740848541259766, -6.406583786010742, -5.367481708526611, -3.0397934913635254, 1.3631333112716675, -8.224811553955078, 14.528425216674805, 18.310192108154297, 3.6375207901000977], [-18.749710083007812, -8.126733779907227, -11.308516502380371, 5.1976423263549805, -14.082490921020508, -11.272489547729492, 10.955470085144043, -2.517848253250122, -5.720319747924805, -8.269598007202148, 23.409517288208008, 17.314455032348633, -2.7176740169525146, -8.72996997833252, 17.581439971923828, 1.8945358991622925, -12.067713737487793, 17.850326538085938, -8.045727729797363, -3.829613447189331, -23.05851936340332, 7.493222236633301, -12.453329086303711, 12.195859909057617, -9.852090835571289, -10.98213005065918, 8.980779647827148, 8.856677055358887, 21.075660705566406, 0.7080572247505188, -6.4478373527526855, 4.816452503204346, -3.2910022735595703], [-3.5699682235717773, -2.407020092010498, -8.360112190246582, -13.735396385192871, 6.379954814910889, -4.213511943817139, -1.9609477519989014, -6.013327121734619, 11.81421947479248, -8.82589054107666, -2.150533676147461, 17.859243392944336, -1.9140832424163818, -18.93865966796875, 11.057802200317383, 2.281707763671875, 4.214356899261475, -28.059524536132812, -10.062898635864258, 2.709655523300171, -4.758953094482422, -6.098235607147217, 13.821640968322754, 4.566410541534424, 6.373791217803955, -20.697208404541016, 14.709500312805176, -12.36973762512207, 11.29374885559082, -2.335125207901001, -11.665986061096191, 4.326594829559326, 13.56201171875], [8.283232688903809, 22.49873924255371, -5.14025354385376, -20.36842918395996, -11.225322723388672, 8.82175350189209, -8.100546836853027, 16.967090606689453, 7.8213725090026855, 3.9227163791656494, -1.499025583267212, 4.407550811767578, -9.323431015014648, 9.463289260864258, -12.013628959655762, -12.178726196289062, -7.612534046173096, 11.082870483398438, 17.032390594482422, -1.6906770467758179, 2.636746406555176, -1.7896801233291626, 16.723342895507812, 29.325611114501953, -5.610430717468262, -17.262828826904297, 2.5404393672943115, 4.311683177947998, 3.908050298690796, -9.311471939086914, -24.0390567779541, 2.022914171218872, 5.75663423538208], [8.699048042297363, 0.7893508672714233, 0.2649571895599365, 10.135878562927246, -0.018908755853772163, 8.033842086791992, -5.744046211242676, 11.888028144836426, 0.3255220949649811, 15.621204376220703, -9.65726375579834, -6.185118675231934, -2.847740888595581, 6.898845672607422, -6.479029655456543, -14.333081245422363, 16.495811462402344, 15.587553024291992, -0.050946783274412155, -13.619669914245605, 23.98164939880371, 15.819703102111816, 8.706927299499512, 2.452685594558716, 1.5116381645202637, 9.458794593811035, 7.724154472351074, 10.679601669311523, -11.147543907165527, -20.279523849487305, -4.338465690612793, 2.715183973312378, 8.785629272460938], [8.333868980407715, -0.21844083070755005, 13.18991756439209, -8.137419700622559, 15.863113403320312, -11.46660041809082, 9.969721794128418, 15.21931266784668, -2.210960626602173, 0.3474007248878479, 11.797064781188965, -6.0608015060424805, -2.157775640487671, -14.743731498718262, -5.5044026374816895, 10.904850006103516, -0.8310338854789734, 9.185501098632812, -10.510862350463867, -7.315404891967773, 4.00424337387085, -11.7424955368042, 11.58934497833252, -4.682925224304199, 4.222856044769287, -9.38696575164795, -1.176987886428833, 3.516087293624878, 4.904257774353027, 0.9059789776802063, 3.5918233394622803, 26.210542678833008, -6.4215545654296875], [-7.105550289154053, -2.303335428237915, -5.080143928527832, 9.958826065063477, -5.271341323852539, -3.825131893157959, -18.326669692993164, 1.6128524541854858, 11.22393798828125, -7.809572219848633, -9.660515785217285, -4.737578868865967, -3.2175331115722656, 11.502099990844727, 7.761624813079834, -1.9703609943389893, 18.85247802734375, 7.805782794952393, -7.3546247482299805, 6.316098213195801, -7.1736884117126465, 17.447399139404297, -15.674241065979004, 7.1391730308532715, 4.597081184387207, 16.934072494506836, -12.018399238586426, 5.922812461853027, -2.8539438247680664, 7.321390151977539, 4.850137710571289, -4.759152412414551, 16.939868927001953], [-1.2096939086914062, 19.373517990112305, -18.773874282836914, -7.401399612426758, -4.829260349273682, -9.163297653198242, -2.676658868789673, -1.404196858406067, 14.894192695617676, -8.911898612976074, 13.103034973144531, -4.681840419769287, 11.266430854797363, -6.822935581207275, -6.034239292144775, 4.208151817321777, -9.022330284118652, -2.2245144844055176, -10.615513801574707, -9.949920654296875, 8.544148445129395, -16.025331497192383, 27.0224609375, -5.091695785522461, -5.8161702156066895, -3.219571113586426, 8.037053108215332, -8.01783561706543, 1.280656099319458, -6.986392498016357, -6.647017478942871, 20.631317138671875, 2.5491578578948975]], "dtype": "float32"}, {"__nd__": [[[-2.5533359050750732, 2.376307725906372, 4.158819675445557, -5.387650966644287, 3.350170135498047, 0.54106605052948, -15.55865478515625, 5.646468162536621, -2.2982394695281982, 1.7220861911773682, 3.3363561630249023, -8.0286865234375, 14.134845733642578, 2.1085619926452637, -9.465192794799805, 6.035473823547363, -6.850477695465088, -8.565964698791504, -6.22806978225708, 1.9949592351913452, -13.247224807739258, 3.216287136077881, -8.906978607177734, 9.511642456054688, -10.072395324707031, -1.268140196800232, 9.434775352478027, 12.529914855957031, 15.887332916259766, 4.016141891479492, 7.375861167907715, -4.860333442687988, -3.1466925144195557], [3.1670994758605957, 2.3710174560546875, 6.020115852355957, 11.67573356628418, -10.415566444396973, -6.9549479484558105, 5.881511688232422, -9.48009204864502, -12.882440567016602, 8.202826499938965, 26.492645263671875, 22.403520584106445, 11.538745880126953, -8.9246826171875, 0.5091627836227417, 6.254071235656738, -8.11587142944336, -10.485434532165527, -13.336389541625977, -5.481669902801514, -20.245555877685547, -1.844042181968689, 2.1858737468719482, -3.969738006591797, 5.206028461456299, 13.068734169006348, 9.839489936828613, 5.951860427856445, 12.71394157409668, 8.3860445022583, -5.436997890472412, -11.204061508178711, -4.35594367980957], [15.553516387939453, 11.88549518585205, 6.828945636749268, -3.234678030014038, 18.971830368041992, 16.88893699645996, -9.151787757873535, -10.055962562561035, 5.811522483825684, -5.862111568450928, -27.16692543029785, -11.50926685333252, 0.8374626636505127, 0.7134836316108704, -2.4238476753234863, 0.7007068991661072, -24.86893653869629, -3.149251699447632, 9.842867851257324, -5.1787238121032715, -2.754544734954834, 4.561978816986084, 19.343887329101562, -0.7215545773506165, 0.22027583420276642, -21.021656036376953, 11.229763984680176, -16.434354782104492, 2.875333547592163, -6.409274101257324, -3.3853566646575928, 3.1499502658843994, -4.402245998382568], [8.688787460327148, -0.5976289510726929, -1.573185920715332, 17.679033279418945, 13.408158302307129, -3.5830459594726562, -2.7293860912323, 0.9019589424133301, -8.4606351852417, -0.9869545102119446, -10.679877281188965, 4.523142337799072, 1.1332085132598877, 3.3984105587005615, -18.99028205871582, 6.458785057067871, -1.98361337184906, 1.0662658214569092, 7.4658708572387695, -0.4150872826576233, -3.8084564208984375, 10.982392311096191, -4.0265984535217285, 4.710488796234131, 1.7191907167434692, 5.536169528961182, -4.376761436462402, -20.50708770751953, 9.702064514160156, -7.387021064758301, -1.1230089664459229, -3.2908132076263428, 0.9472535252571106], [14.546524047851562, 16.92873191833496, -4.432495594024658, 15.988910675048828, -17.951284408569336, -5.418204307556152, 2.077317953109741, -1.8903089761734009, 10.701444625854492, 2.086638927459717, 13.57225513458252, -12.54122257232666, 12.884173393249512, 8.623908042907715, 2.719996452331543, -9.805880546569824, 2.681321859359741, -13.040056228637695, 0.8276491165161133, 7.493785381317139, 2.9782636165618896, -12.086347579956055, 15.717586517333984, 21.720600128173828, -16.25220489501953, -3.876286268234253, -2.0799922943115234, 28.869216918945312, -6.300133228302002, -1.8846075534820557, 4.396486282348633, -3.6914191246032715, 0.695105254650116], [14.793920516967773, 5.744931697845459, 1.6086031198501587, -10.929614067077637, -15.994651794433594, 2.1635637283325195, 9.834959983825684, -1.9661431312561035, 7.782419681549072, 15.228660583496094, 13.46396541595459, -4.83815336227417, 4.652315616607666, -6.38023042678833, 13.80267333984375, 1.5184810161590576, -1.3917322158813477, 1.5099064111709595, -6.993489742279053, 4.478943347930908, -13.085470199584961, -16.902755737304688, 5.225970268249512, -2.8609516620635986, 3.982391834259033, -20.208616256713867, 5.263352870941162, 25.38666343688965, 9.202675819396973, 6.811099529266357, 6.935202121734619, 3.691455841064453, 0.355153888463974], [-5.625328540802002, -1.1465295553207397, 6.831356048583984, 17.869396209716797, 9.507173538208008, -1.0673131942749023, -21.08803939819336, -1.3326960802078247, 2.268869400024414, 3.1050162315368652, 14.276010513305664, 5.255858421325684, -2.0740883350372314, -7.753734588623047, 6.221418857574463, 11.309836387634277, -5.235614776611328, -2.644352912902832, -2.5202078819274902, 2.479679822921753, -11.822101593017578, 9.585956573486328, -6.164267539978027, -1.2033919095993042, -11.71487808227539, -7.815042972564697, -9.155884742736816, 6.7340989112854, -3.7053184509277344, -2.1243183612823486, -1.5225869417190552, 1.457674503326416, 14.758811950683594], [-4.119194030761719, -5.549951076507568, -6.779970645904541, -10.070860862731934, 9.46738052368164, -3.624626636505127, -10.588935852050781, 3.3999218940734863, 5.673383712768555, -13.524670600891113, -1.1839525699615479, -3.230224847793579, 9.212512016296387, -3.909952402114868, 4.320709228515625, -8.020280838012695, -6.056773662567139, 4.160272121429443, 5.048281669616699, 10.572543144226074, -10.536293029785156, -5.991170883178711, -9.833279609680176, -2.5760858058929443, -3.216407537460327, 11.545294761657715, -8.104310989379883, -12.601652145385742, 1.1727133989334106, 7.7175612449646, -7.943735599517822, 8.422996520996094, 7.142762184143066], [-18.3039608001709, -3.2229185104370117, 4.32546329498291, 3.7086498737335205, 2.04489803314209, 6.617099761962891, -4.278393745422363, -1.0420690774917603, 9.333304405212402, -14.58198356628418, -5.256403923034668, -8.213592529296875, 11.806469917297363, 25.59890365600586, -11.459216117858887, 8.422807693481445, 10.141640663146973, 1.1569161415100098, -8.37585163116455, -12.438172340393066, 28.842519760131836, -10.934566497802734, -1.5507068634033203, 0.7616581320762634, 3.4878969192504883, 2.4184417724609375, -14.601314544677734, -15.135017395019531, 1.6132229566574097, 1.911062479019165, 2.159370183944702, -5.994506359100342, -13.378252983093262], [-2.8901586532592773, 0.44113069772720337, 12.734526634216309, -5.6915788650512695, 0.8646745085716248, -3.8220694065093994, -6.131150722503662, -1.9874976873397827, 14.674525260925293, 6.437786102294922, -18.719438552856445, -4.837345123291016, -3.8429572582244873, -1.3736741542816162, -9.466672897338867, -6.083317279815674, 13.96056842803955, 4.721275806427002, -3.3451180458068848, -11.829906463623047, -11.603915214538574, -14.93215274810791, -20.995534896850586, -4.370548725128174, 15.035957336425781, 10.287580490112305, 0.6951668858528137, -0.6007880568504333, -8.218389511108398, -16.071063995361328, -5.585448741912842, 2.487881898880005, -2.4604225158691406], [-11.174638748168945, 9.598731994628906, -18.056068420410156, -2.4739186763763428, -2.9779117107391357, -8.153057098388672, 10.127436637878418, 2.1724867820739746, 9.801244735717773, 0.8883997201919556, 0.23585152626037598, -4.254331588745117, 11.621851921081543, -2.8894381523132324, -14.850895881652832, -3.4496102333068848, 11.909703254699707, 6.409300804138184, -1.9981154203414917, 8.248322486877441, 9.664495468139648, -0.8791735768318176, -10.37824535369873, -21.055803298950195, -4.432592868804932, 1.4818145036697388, -17.142301559448242, -6.145442485809326, -7.156051158905029, 1.0821142196655273, 5.310659408569336, 0.7897792458534241, -0.32711270451545715], [2.1643199920654297, -3.6254501342773438, 16.377037048339844, 5.532244682312012, 17.64346694946289, -1.7882530689239502, -2.3484768867492676, 5.559385776519775, -1.6352007389068604, 2.8354299068450928, -12.545328140258789, 1.4320493936538696, 1.6309804916381836, -3.954533100128174, -8.72969913482666, 10.792632102966309, 5.876189708709717, -10.802732467651367, -15.017363548278809, 14.079167366027832, -7.913140773773193, -0.5579593181610107, -3.4980416297912598, 11.076813697814941, -0.3457767367362976, 8.155730247497559, -10.96874713897705, 14.326184272766113, 4.492353916168213, -3.1068027019500732, 7.072631359100342, 5.581096172332764, 10.947208404541016], [-5.525501251220703, 9.061949729919434, 4.085198879241943, -6.625133991241455, 18.406461715698242, 14.109713554382324, -1.199859380722046, 16.087038040161133, 5.461844444274902, 2.767395257949829, -10.060858726501465, 6.5604376792907715, -19.85756492614746, 12.688667297363281, 1.414589285850525, 12.425056457519531, -18.228721618652344, -4.436062812805176, 8.178977012634277, -2.800931692123413, -5.885331630706787, 6.315826892852783, 5.669284343719482, 9.247846603393555, 6.803224086761475, -17.159847259521484, 2.319396734237671, 5.334746837615967, 2.162376880645752, 1.6155035495758057, 7.877479076385498, -9.238554000854492, -5.705545425415039], [0.8745644688606262, 9.710371017456055, 0.7109085917472839, -6.66072940826416, 8.572362899780273, 15.725631713867188, 11.46670913696289, -3.964057207107544, -14.516075134277344, 7.16591215133667, 2.445744037628174, -2.463953733444214, -8.955962181091309, -13.386330604553223, 4.114889621734619, 1.6848347187042236, -5.980319976806641, -11.485754013061523, 6.9043049812316895, 5.004263877868652, 15.057161331176758, 7.119293212890625, -1.2420693635940552, -16.27634048461914, 1.8028333187103271, -8.575156211853027, 10.153044700622559, -17.968355178833008, -4.06013822555542, -5.07307767868042, -1.8522809743881226, 9.695710182189941, 15.092557907104492], [12.248858451843262, 7.308432102203369, -4.748076915740967, 6.022978782653809, -7.975097179412842, 5.04763650894165, -1.2685872316360474, -2.1300408840179443, -4.005895614624023, 5.377450466156006, 6.4230217933654785, -21.77633285522461, 7.254096031188965, -10.149816513061523, 0.6392231583595276, 7.126636028289795, -5.024255752563477, 0.7016529440879822, -11.880576133728027, 11.631586074829102, -10.318195343017578, -2.351940870285034, -0.39990031719207764, -6.551328182220459, -3.1981279850006104, -5.226449966430664, 0.256046324968338, 6.7443976402282715, 7.330592632293701, 2.160501480102539, -6.464979648590088, -13.583263397216797, 3.667804002761841], [-1.2042452096939087, 9.770768165588379, 7.973456859588623, 4.414280414581299, 3.5246307849884033, 0.07604427635669708, -0.9775245785713196, 0.3977929651737213, 6.882421016693115, -2.539886951446533, -6.416387557983398, 5.867394924163818, 2.7291479110717773, -9.223448753356934, 24.23553466796875, 1.8771458864212036, 6.773036479949951, -1.5054365396499634, -1.1581753492355347, 3.384870767593384, -8.660646438598633, -2.323683023452759, -9.22364616394043, 0.7899594902992249, 5.936355113983154, -2.1661107540130615, 9.5199556350708, 13.580460548400879, -4.249067306518555, -8.18371868133545, -10.811058044433594, -14.439818382263184, -5.23206090927124], [3.4354441165924072, 10.069501876831055, 2.040356397628784, -9.392751693725586, 2.992370367050171, 5.8254828453063965, -9.690698623657227, -9.306097984313965, -2.2389304637908936, 9.503495216369629, 0.9929916858673096, -1.2092360258102417, 0.0068695079535245895, -2.1309731006622314, 9.736350059509277, 8.454873085021973, 2.4157392978668213, -7.634143352508545, 5.160991191864014, -1.7942489385604858, 1.7449349164962769, 0.6520074009895325, -4.403895854949951, -16.304346084594727, 1.795443058013916, 2.7438488006591797, 7.662039279937744, -12.899404525756836, 4.129814624786377, -0.23816357553005219, -0.03638513386249542, 4.664341926574707, -6.971473217010498], [-8.60354995727539, -22.99846076965332, 1.4884450435638428, -10.634939193725586, -8.426876068115234, 2.904975652694702, -16.003734588623047, -7.6619086265563965, 13.906614303588867, -2.894336223602295, -21.7677001953125, 13.948204040527344, 6.942611217498779, 3.024021863937378, 10.81803035736084, -7.019597053527832, 21.153209686279297, 7.8274993896484375, -3.868393659591675, -13.241415023803711, -8.38243579864502, -0.3343246281147003, -4.547707557678223, 1.9241914749145508, 15.149365425109863, 3.4827301502227783, 1.3575248718261719, -5.7218427658081055, 4.239595413208008, 0.1665036827325821, 0.24111437797546387, -1.0388020277023315, 8.53930377960205], [3.1782262325286865, -3.2667956352233887, 5.023776531219482, 9.190165519714355, -4.4520487785339355, -3.1401829719543457, -8.608671188354492, -19.140928268432617, 10.290607452392578, 0.38686323165893555, -3.3127901554107666, 14.535941123962402, 5.144766807556152, 7.456943511962891, -1.211304783821106, -4.963419437408447, 0.34198030829429626, 12.369847297668457, -4.018460750579834, 6.120842933654785, 2.624471426010132, 7.326602935791016, 10.804974555969238, 3.109726905822754, 4.785889148712158, 2.2517507076263428, -9.469432830810547, 4.416167259216309, -3.0022644996643066, -7.650050640106201, 6.339076042175293, 9.111705780029297, -3.997260332107544], [1.572914481163025, 7.664958477020264, 0.3336975574493408, 2.6838090419769287, 5.877224445343018, 4.754132270812988, 1.2452731132507324, 13.033673286437988, 1.4916014671325684, 9.464348793029785, -28.439903259277344, -0.17164307832717896, 3.157719612121582, 2.1901376247406006, -15.458910942077637, 7.664479732513428, -9.683161735534668, 6.674158573150635, -6.341182231903076, -0.7400703430175781, -13.580949783325195, 9.670524597167969, -3.194791316986084, 6.225674152374268, 6.4763712882995605, -4.8263163566589355, 10.520537376403809, 4.456875801086426, 0.31823620200157166, -1.3929470777511597, -14.38572883605957, -2.787169933319092, 3.400393009185791], [6.38485050201416, 3.1167571544647217, 2.083752393722534, 1.7710251808166504, 4.2928009033203125, 4.057568550109863, -0.8833773732185364, 11.133546829223633, -10.94560432434082, 14.680360794067383, 11.64887523651123, 9.343833923339844, 9.419526100158691, 5.2944746017456055, 1.7360508441925049, -16.34445571899414, -0.8492863178253174, 12.661384582519531, -3.9577293395996094, -0.6331171989440918, -1.4091169834136963, 12.804551124572754, 8.559830665588379, 2.4827163219451904, 11.545658111572266, 2.0849125385284424, -6.694766521453857, -8.159303665161133, 12.073967933654785, 13.309073448181152, -32.11318588256836, -2.9609320163726807, -2.4513351917266846], [-7.368203639984131, 19.120283126831055, 1.121246099472046, -13.451663970947266, -1.7920771837234497, 15.992114067077637, -5.787957191467285, 10.66393756866455, 11.510544776916504, 16.295194625854492, 6.10911226272583, -10.341703414916992, -1.516594409942627, -6.378282070159912, 4.719111919403076, 0.9773894548416138, -4.633456707000732, -11.663968086242676, 3.3485820293426514, -14.147857666015625, 1.475533127784729, -4.059427738189697, -1.1860262155532837, -9.497720718383789, 3.4258646965026855, -3.5092198848724365, 1.867497444152832, 2.5107600688934326, -1.2385917901992798, -9.183998107910156, 7.121031284332275, -13.76270580291748, 1.9913914203643799], [11.119866371154785, 7.1704230308532715, -10.361406326293945, -0.686162531375885, -5.785367965698242, -4.950196266174316, 3.7943716049194336, 6.907231330871582, 0.09296946972608566, -3.3813817501068115, 0.198710635304451, 10.97378921508789, 6.596445083618164, 22.955392837524414, 15.478740692138672, -1.083211898803711, -4.790051460266113, 30.41570281982422, -0.7065421938896179, 7.846746444702148, -6.223433494567871, 11.955842971801758, 11.69511604309082, -1.8102654218673706, -2.4536619186401367, -8.790040969848633, -13.501352310180664, -6.130622386932373, -0.4568178653717041, 9.303189277648926, -3.523550271987915, 0.6323397755622864, -2.6262476444244385], [-12.067730903625488, -5.20958137512207, 13.07221794128418, 10.567320823669434, 13.44503402709961, -5.155946731567383, 1.3913503885269165, -8.338789939880371, -5.652909278869629, 8.619804382324219, 12.880495071411133, 4.241666316986084, 5.402981758117676, -10.167620658874512, 1.3637269735336304, -4.779787540435791, 12.155696868896484, -11.030834197998047, -1.8525946140289307, -3.8112523555755615, -3.135784387588501, -1.786911129951477, 3.9819397926330566, -2.2915072441101074, -9.771778106689453, 18.205232620239258, 14.053346633911133, -16.826370239257812, 8.526139259338379, 15.76814079284668, -12.534002304077148, -7.739938259124756, -14.817258834838867], [7.274502754211426, 8.638358116149902, 8.000396728515625, -0.7861294150352478, -2.919888496398926, 12.193567276000977, 7.080046653747559, -5.154373645782471, -15.4169921875, 6.796077251434326, 5.4406023025512695, 2.158478021621704, -4.423746585845947, 2.3639938831329346, -9.508917808532715, -2.064185619354248, -13.08009147644043, -7.573981285095215, 4.560726642608643, -19.103960037231445, 5.812782287597656, 3.190647602081299, 13.406024932861328, -3.985278367996216, 4.165372371673584, 2.1304945945739746, -0.3156720697879791, -5.945118427276611, 5.031466960906982, -14.646109580993652, -1.0287851095199585, 6.036944389343262, -7.16140604019165], [-5.827799320220947, 0.6586341857910156, -9.464723587036133, -0.7684403657913208, 7.886432647705078, 5.902996063232422, 14.323220252990723, 8.736080169677734, -10.250397682189941, -9.019048690795898, 9.824021339416504, -6.888692855834961, 10.057868003845215, -6.845757484436035, 5.913793563842773, 12.412447929382324, 9.3297758102417, -4.461243629455566, -24.286516189575195, -5.480051517486572, 0.6542090177536011, -9.908540725708008, 5.146363735198975, -1.994733214378357, -7.9526896476745605, 9.125568389892578, -7.427341461181641, -4.876379013061523, 11.167313575744629, -1.9973647594451904, 10.66219711303711, 10.6039457321167, 5.322447299957275], [5.747475624084473, -16.39846420288086, -8.62120246887207, -4.3979363441467285, -1.5796749591827393, 0.7516889572143555, -14.192666053771973, -19.806379318237305, 25.606029510498047, -13.250534057617188, -8.039182662963867, 6.937606334686279, 0.004807461518794298, -0.023358216509222984, 8.764358520507812, -3.3068439960479736, 8.390803337097168, -0.9247121810913086, 7.976019859313965, -12.124966621398926, -7.559514045715332, 2.2892653942108154, 7.345030784606934, 0.23341049253940582, -5.657140731811523, -10.891802787780762, 6.126299858093262, -6.030829906463623, 2.524956226348877, 17.168615341186523, -6.560544967651367, 4.50048828125, -2.439389228820801], [0.7539047598838806, -3.0852818489074707, -6.240669250488281, -7.386722087860107, -10.616119384765625, -4.211084365844727, -6.003283977508545, -1.2086726427078247, -0.2677905857563019, 10.851861953735352, -8.186820030212402, 9.905445098876953, 6.861054420471191, -3.2635889053344727, -0.9511223435401917, -2.5344173908233643, -1.9278146028518677, -1.9311717748641968, 9.676093101501465, 11.592195510864258, -6.014930725097656, 1.9986815452575684, -19.54771614074707, 3.3305907249450684, 5.692677021026611, 2.395329236984253, -1.396485686302185, 16.014827728271484, -7.97754430770874, -5.908846378326416, 10.767680168151855, 7.367044925689697, 22.92574119567871], [-6.750504970550537, -27.58517837524414, -9.125434875488281, -8.55708122253418, -2.515324354171753, 1.227768898010254, -12.04016399383545, -9.891912460327148, 1.1748168468475342, -4.5925374031066895, 5.5226359367370605, 9.655975341796875, -4.103324890136719, -14.518847465515137, 9.6266450881958, -1.5579763650894165, 6.660365104675293, -7.793369770050049, -12.186782836914062, 5.901346206665039, 4.792733669281006, 2.581374406814575, -1.310094952583313, -3.829373359680176, -0.29542067646980286, -3.321500062942505, 3.3339664936065674, -18.03053092956543, 5.925170421600342, -1.7167166471481323, -9.535804748535156, -1.1820131540298462, 1.7944093942642212], [0.568774938583374, -13.699678421020508, -3.257537364959717, -6.85989236831665, 7.6326799392700195, -4.810858726501465, 7.132650852203369, 10.06541919708252, 10.517815589904785, 6.213000774383545, -1.4443023204803467, -8.609277725219727, -2.4074294567108154, 8.089980125427246, 0.7163439989089966, -7.57125186920166, 7.057666778564453, -3.9102237224578857, 1.113806128501892, -3.1651992797851562, -20.681217193603516, -11.537768363952637, 3.0326662063598633, 11.783102035522461, -8.565114974975586, -6.346078872680664, -5.881048679351807, 5.876871585845947, 3.1890201568603516, 11.473638534545898, -2.6154000759124756, 15.423344612121582, 3.671426296234131], [1.4966795444488525, 20.345355987548828, 7.539578437805176, 9.125232696533203, 1.241514801979065, 3.3263700008392334, -12.931681632995605, 2.930757522583008, 8.484541893005371, -2.5952157974243164, -13.395803451538086, -1.7953195571899414, 7.24277400970459, 13.58389949798584, 2.7004477977752686, 19.860565185546875, -10.574520111083984, 5.683395862579346, -6.453581809997559, -13.408663749694824, 14.596817016601562, 4.558448791503906, 14.330281257629395, -7.34940242767334, 2.7195634841918945, 2.6341006755828857, -17.93671226501465, 3.1114466190338135, -0.10929559916257858, -4.689584732055664, -5.554664611816406, -4.804200649261475, -5.990081310272217], [-11.369163513183594, -15.815948486328125, -4.832313537597656, -1.826621651649475, -2.277902603149414, -2.982217788696289, 6.169034481048584, -2.215898036956787, -1.7572836875915527, 5.4652099609375, -12.655652046203613, -4.154873371124268, 14.454790115356445, -15.102606773376465, 0.1517697274684906, -5.950587749481201, -5.201563358306885, 4.413183689117432, -9.354202270507812, 17.86272621154785, 15.470982551574707, -4.007817268371582, -22.432464599609375, 7.141499996185303, 7.697237014770508, 2.3311550617218018, -11.582843780517578, 7.105252265930176, 0.6006434559822083, -4.847771644592285, 12.34939193725586, 13.02327823638916, 10.707430839538574], [5.647952556610107, -3.345942497253418, 6.655744552612305, -4.044651508331299, 3.1953694820404053, -16.4609432220459, 12.90893268585205, -9.078524589538574, -1.8602005243301392, -9.025964736938477, 4.644054889678955, -19.317367553710938, -6.949947834014893, -28.984081268310547, -8.460127830505371, 6.005648136138916, 6.906357765197754, -14.482958793640137, -11.54299259185791, 3.8046762943267822, -7.780452728271484, -19.900768280029297, -2.854310989379883, -6.06441593170166, -11.14772891998291, 2.1487302780151367, -7.563220024108887, -3.662734031677246, -0.826852023601532, -12.845916748046875, 3.9416520595550537, 4.798178672790527, -8.299959182739258], [10.286787986755371, -6.367402076721191, -1.8984310626983643, 14.638153076171875, 1.8796923160552979, -0.06720918416976929, 12.152460098266602, -3.968698501586914, 10.68709659576416, -9.19870376586914, -7.369001388549805, 12.516999244689941, 9.566062927246094, 3.048353672027588, 2.3966894149780273, -10.082662582397461, -3.7333903312683105, 16.84530258178711, 9.060009002685547, -8.271586418151855, 10.267806053161621, 0.5044971108436584, -5.020737648010254, -10.463543891906738, 7.964138984680176, 9.327322006225586, -22.42608070373535, 5.211913585662842, -11.157773971557617, -6.441296577453613, -3.732034683227539, -5.43109655380249, -1.1328110694885254], [-15.869486808776855, -19.45226287841797, 1.4641848802566528, 7.958607196807861, -5.0122222900390625, 2.8626296520233154, 8.409459114074707, 16.159618377685547, -0.0362916961312294, -5.3863139152526855, 1.7797455787658691, 2.3517422676086426, -8.781131744384766, 13.344024658203125, -1.5744048357009888, -1.4576243162155151, 3.546252727508545, 0.10059460252523422, -5.514991760253906, -11.470846176147461, 14.946073532104492, -11.177078247070312, 3.9527273178100586, -10.880887985229492, 0.43915122747421265, 16.040424346923828, -1.2290109395980835, -8.351627349853516, -8.97578239440918, 0.9462798833847046, -2.76831316947937, -8.728564262390137, -14.224678039550781], [3.192525863647461, 6.998249530792236, -13.623709678649902, 1.932011604309082, -6.346342086791992, -1.2869789600372314, 2.2045817375183105, 6.728833198547363, -0.2046612948179245, -0.504043698310852, 10.109134674072266, 5.403580188751221, 2.949291706085205, 1.4088610410690308, 4.384354591369629, -6.442668437957764, 3.0416064262390137, 1.2824121713638306, -4.235743522644043, 19.62179946899414, 11.298236846923828, 0.3131283223628998, -4.626235008239746, 7.716014862060547, 5.168043613433838, 0.23207087814807892, -9.820552825927734, 1.6338779926300049, -6.148799896240234, -3.894909620285034, 16.401227951049805, 5.498247146606445, 2.338526725769043], [-4.9326910972595215, -5.501094341278076, -4.505979061126709, 0.19513453543186188, -6.232021808624268, -15.160725593566895, 3.7706081867218018, 0.03518994525074959, 10.534327507019043, -16.938241958618164, -17.96681022644043, 3.4503846168518066, -16.95180320739746, 4.895595550537109, 2.1634247303009033, 0.6492510437965393, 22.964111328125, 1.580879807472229, -8.817309379577637, 5.903845310211182, 10.899153709411621, 2.518583297729492, -26.214704513549805, -12.415825843811035, 18.651878356933594, -8.621139526367188, -4.663049697875977, -9.59653091430664, -5.824811935424805, 12.79124927520752, 5.805335521697998, 5.368192195892334, 4.666903495788574], [-5.666177272796631, 1.7122061252593994, 8.7933988571167, 5.484974384307861, 10.435158729553223, 4.6389689445495605, 2.1501541137695312, 9.938023567199707, 3.4877352714538574, 14.057123184204102, -7.951762676239014, 2.6682913303375244, -3.894554615020752, 15.232278823852539, -19.564857482910156, -3.6891679763793945, -8.74385929107666, -0.218521386384964, -1.3018214702606201, -4.2532548904418945, 5.651930809020996, 19.569671630859375, 12.74386978149414, 4.268626689910889, -14.996882438659668, -2.4549574851989746, 1.9729050397872925, 2.1682019233703613, 0.4983600378036499, -16.636043548583984, -9.918314933776855, -12.222021102905273, -19.254404067993164], [2.2290918827056885, 1.3158355951309204, 3.4903862476348877, -2.3802576065063477, -2.691956043243408, 4.8465046882629395, -1.115320086479187, 6.797186374664307, 8.361865997314453, -5.707522392272949, 12.434700965881348, 4.38800573348999, -8.501704216003418, 9.589468002319336, 6.112384796142578, 4.504342555999756, -7.842596054077148, -3.2439451217651367, -2.270261287689209, -9.060563087463379, 2.080874443054199, 6.327858924865723, 11.291962623596191, -15.741663932800293, 0.7036828398704529, -1.7161089181900024, -9.188750267028809, 6.588330268859863, -2.212620496749878, -3.785393476486206, -0.8390778303146362, -1.4324514865875244, -7.157860279083252], [-4.83646821975708, -6.875319957733154, 4.905638217926025, 8.622065544128418, 11.80392074584961, 21.716726303100586, -0.6691557765007019, 1.5241717100143433, -9.738340377807617, -2.4601826667785645, -14.404473304748535, 2.300140380859375, -4.081161975860596, 2.6863973140716553, -4.526041030883789, 1.7918381690979004, -2.739245891571045, 0.8598539233207703, 2.6927356719970703, -7.027210235595703, 11.48063850402832, -2.4028286933898926, -18.545406341552734, -12.958638191223145, 11.201796531677246, -1.1366221904754639, -3.4633727073669434, -12.887070655822754, -2.5803678035736084, -11.569116592407227, 15.876500129699707, 5.581143379211426, 8.87503433227539], [0.7153472304344177, 1.440312147140503, 4.663032054901123, 4.330256462097168, 2.17169451713562, 11.01721477508545, -9.878270149230957, -7.862377166748047, 9.659814834594727, -5.4782562255859375, 1.9959542751312256, -2.1103878021240234, 8.611849784851074, 1.1832877397537231, -0.04790285974740982, 8.856902122497559, -5.365137577056885, -12.344959259033203, -8.738055229187012, -9.880693435668945, 12.729549407958984, -0.07294190675020218, 8.618881225585938, -7.474910736083984, -5.858401298522949, 1.8952810764312744, 15.029642105102539, -8.474761962890625, -6.227097034454346, -4.741512298583984, 11.335527420043945, -4.5944600105285645, 3.255535125732422], [1.244411587715149, -4.854913711547852, 6.1355695724487305, 5.557904243469238, -0.554643988609314, -8.437446594238281, -5.7765793800354, 5.82112455368042, -2.941051721572876, -4.357417583465576, 1.5391006469726562, -5.157350063323975, -10.537431716918945, -2.2130684852600098, 10.287137985229492, 1.8377503156661987, 0.5564867854118347, 1.457932710647583, -6.13567590713501, 1.218692660331726, -16.71349334716797, -6.864719390869141, -11.470965385437012, 5.081885814666748, 4.720158576965332, 19.840892791748047, -9.781442642211914, -1.804487943649292, 3.2947628498077393, 9.127361297607422, -9.242508888244629, 2.292316198348999, 7.4045305252075195], [-8.283172607421875, -9.392765998840332, 10.408136367797852, -17.653745651245117, -0.7369025349617004, 11.588379859924316, -2.736023426055908, 6.753796577453613, 2.2208914756774902, -3.3819949626922607, -4.87796688079834, -22.128923416137695, -7.075568199157715, -1.0439183712005615, -21.10257339477539, -7.224441051483154, -20.850488662719727, 1.7663347721099854, -7.622910976409912, -7.182993412017822, -0.9227232933044434, -0.657077431678772, 0.6419374942779541, -1.5490413904190063, 2.2402517795562744, -6.105902194976807, 1.7066457271575928, 13.312313079833984, 10.079644203186035, -14.572990417480469, 10.953665733337402, -5.827908039093018, 7.675520896911621], [9.73587417602539, 3.006757974624634, -10.754056930541992, -11.424994468688965, -4.408666610717773, 0.552763819694519, 2.6260080337524414, 15.907806396484375, -20.58051109313965, -0.7500109672546387, 10.972800254821777, 3.2352097034454346, 13.580039024353027, 0.8718127608299255, -1.7241156101226807, -13.713577270507812, -4.649986743927002, 10.830388069152832, 11.89194107055664, 2.420247793197632, 4.7115912437438965, 9.740878105163574, -2.042863130569458, -4.727025985717773, 0.4381491541862488, 14.79912281036377, -6.3415679931640625, -2.686509847640991, -5.936151504516602, -4.452180862426758, 0.8703036308288574, 2.0882585048675537, -11.225762367248535], [4.873929023742676, 2.38092303276062, -8.500107765197754, -3.7608883380889893, -9.881836891174316, -6.401444911956787, 3.8229496479034424, -8.423053741455078, -4.345242023468018, 7.1337785720825195, 18.356809616088867, 4.06862211227417, -0.3977202773094177, 10.415816307067871, -10.737649917602539, 10.794525146484375, -3.7671821117401123, -1.6034183502197266, -9.497377395629883, 1.1167712211608887, -1.3298455476760864, -8.0531644821167, 8.365180969238281, -4.116618633270264, -3.1603126525878906, -8.502197265625, 6.633040428161621, 7.570282936096191, 6.475996017456055, 5.171137809753418, 1.6378880739212036, 7.183375358581543, 0.04196072369813919], [10.87544059753418, -6.494814872741699, -3.8893487453460693, -3.5604681968688965, -1.3663933277130127, -7.3851399421691895, -7.644225120544434, -11.569875717163086, 11.970870018005371, -6.243534564971924, -10.477458000183105, 11.649450302124023, 1.451732873916626, -5.747762680053711, 19.626834869384766, 3.0104293823242188, -3.1268272399902344, -0.1337955743074417, 8.204824447631836, 10.955621719360352, -20.536094665527344, -13.033365249633789, -3.0466554164886475, 0.019201356917619705, -4.445096969604492, -5.399239540100098, -2.3857827186584473, 13.145699501037598, -10.148209571838379, -2.012406349182129, -5.436868667602539, 3.0540664196014404, 3.9526150226593018], [-0.3968338966369629, -11.613777160644531, 8.998411178588867, -0.7128344774246216, -4.96095085144043, -0.8491601943969727, -8.044135093688965, 0.9574316143989563, 13.273555755615234, 0.594265878200531, -5.131074905395508, -6.020502090454102, -18.15532875061035, -7.031327724456787, -10.974266052246094, -11.417316436767578, 4.315321445465088, 1.4617542028427124, -4.532559871673584, 22.380844116210938, 10.555890083312988, -11.631253242492676, -4.270651817321777, 7.61862325668335, 4.130465507507324, 8.502185821533203, 7.511280536651611, -2.315443515777588, -15.481155395507812, 5.062954425811768, -8.120869636535645, -10.807377815246582, 2.092836856842041], [-15.237568855285645, -5.8918046951293945, 4.343006134033203, -2.4937474727630615, 10.058688163757324, 13.171979904174805, 5.015181064605713, -3.0147879123687744, -3.247849225997925, 13.27542495727539, -12.138590812683105, 3.4621524810791016, -9.905715942382812, -1.4541133642196655, 5.492258548736572, 0.20175789296627045, -8.530936241149902, -7.560479164123535, -6.262814044952393, 8.698569297790527, 14.88090705871582, -1.2465767860412598, 2.674947500228882, 0.04634305462241173, -7.9806647300720215, 1.0855578184127808, 8.172691345214844, -15.83584213256836, 10.178092002868652, -13.104665756225586, -12.607406616210938, 12.658833503723145, 5.541590213775635], [23.621540069580078, 27.568941116333008, 9.790862083435059, -20.616756439208984, -7.426945686340332, 5.278416156768799, 2.237874984741211, 2.57637095451355, -2.710785388946533, 3.2200074195861816, -17.295162200927734, -3.8629982471466064, -0.5517240762710571, -11.895289421081543, -23.82346534729004, 5.876859188079834, 3.6368775367736816, 12.14797592163086, 3.020888090133667, -15.984305381774902, 9.077079772949219, -5.796789169311523, 20.753055572509766, 0.0318252295255661, 0.9339136481285095, -2.2463228702545166, -7.108166694641113, 7.3289103507995605, -11.2505521774292, -1.567511796951294, -9.132136344909668, -2.141172409057617, -1.294519305229187], [-0.6481810808181763, -2.4542956352233887, -8.331123352050781, 10.424610137939453, -5.038609027862549, 3.8070836067199707, -5.1018900871276855, 2.3727924823760986, 5.775715351104736, -3.428708791732788, 9.621735572814941, -9.71386432647705, -9.013434410095215, -5.9229583740234375, -0.7793999910354614, -8.802315711975098, 6.019497871398926, 3.274435520172119, 2.986455202102661, -3.7695071697235107, 0.5338394641876221, -13.515488624572754, 6.183384895324707, -7.7513957023620605, -17.861570358276367, 10.065263748168945, 9.66181755065918, -1.4313610792160034, -13.74350357055664, -1.173875093460083, 7.237321376800537, -17.649682998657227, -2.704256296157837], [0.6136298775672913, 2.736743927001953, -4.272006034851074, 0.258456826210022, 9.804586410522461, 0.10144226253032684, -1.763895034790039, -4.091984748840332, -12.86213207244873, -9.815418243408203, -13.066036224365234, 4.370344161987305, -12.938148498535156, 21.831256866455078, -2.140415906906128, 8.464704513549805, 3.211491823196411, 12.142538070678711, -5.037700176239014, -2.9258573055267334, 2.9263548851013184, 8.097553253173828, 13.700013160705566, -8.737044334411621, 10.89541244506836, 6.219034194946289, -1.4594045877456665, -13.008295059204102, -2.275818347930908, 2.606231212615967, -1.8566004037857056, -2.717055559158325, -6.948587894439697], [-5.9481000900268555, -20.008464813232422, -13.970909118652344, -4.0147809982299805, 14.49997615814209, 5.6276679039001465, 5.88240385055542, -3.0829689502716064, -12.177199363708496, -11.243247985839844, 0.20114301145076752, 2.9050791263580322, -3.546907424926758, -2.6556034088134766, -2.7456400394439697, 12.960397720336914, 9.845829963684082, -0.20106588304042816, 4.011934757232666, 3.7046358585357666, 4.270534992218018, -6.9051079750061035, -4.137187957763672, -20.971004486083984, -17.695096969604492, -1.920261263847351, 9.706672668457031, -14.626340866088867, -5.166842460632324, 0.7538532018661499, 24.155447006225586, 8.77389907836914, 1.221806287765503], [-7.335077285766602, 8.694868087768555, 8.268335342407227, 11.76536750793457, 6.460838317871094, 8.143348693847656, -0.5994648933410645, 10.111519813537598, -0.21057270467281342, 15.439384460449219, -0.9226685166358948, 6.923403263092041, 0.8261244297027588, -2.25654935836792, -0.6999020576477051, -6.869537353515625, 3.1584086418151855, -10.106385231018066, -9.67342472076416, -2.754467487335205, -2.106738567352295, -0.05118633061647415, -7.773403644561768, -2.38921856880188, 10.004748344421387, 9.215469360351562, 2.8120017051696777, 8.89067268371582, -6.070084571838379, 5.068410396575928, -7.744421482086182, 13.32601547241211, -2.7621800899505615], [-5.1505279541015625, 25.52634048461914, 2.6937572956085205, 2.321040153503418, -3.063385248184204, -0.4447343051433563, -14.001116752624512, 4.792634010314941, -15.990501403808594, -0.26662304997444153, -1.6163498163223267, -17.881587982177734, -4.009732723236084, 5.402215003967285, -9.369548797607422, 11.322381019592285, 0.7093002796173096, 0.8431424498558044, -10.151630401611328, 3.726515054702759, -3.483982563018799, 4.393728733062744, -6.762667655944824, 8.60803508758545, 2.22967529296875, 8.632139205932617, -4.274284362792969, 17.84609031677246, 6.4776153564453125, 7.615094184875488, 9.869418144226074, -5.880626201629639, -4.74105167388916], [0.5511422753334045, 9.579916954040527, 3.826228380203247, 1.2144628763198853, -14.756134033203125, -1.1525195837020874, 6.296498775482178, 2.0604140758514404, -9.072240829467773, 0.42936038970947266, -14.583489418029785, -4.982654094696045, -14.561382293701172, -6.950562953948975, -4.144906520843506, 14.505057334899902, 11.555819511413574, -0.28252583742141724, 12.396642684936523, -1.4558594226837158, 7.671374797821045, 6.1138176918029785, 2.639472484588623, 6.118998050689697, 1.6471883058547974, 8.96835708618164, 0.6264823079109192, 1.9252647161483765, -2.0054101943969727, -4.525112152099609, -19.777389526367188, -1.8693970441818237, -15.785845756530762], [-5.010444164276123, 10.990992546081543, -7.467331886291504, -16.586755752563477, -2.1194097995758057, 4.860495090484619, -14.475533485412598, 5.409396648406982, 14.901775360107422, -7.061595916748047, -30.63356590270996, -15.254926681518555, -6.947419166564941, -1.2461965084075928, -4.227112293243408, -3.291118621826172, 4.425304412841797, -5.319648742675781, 19.40951919555664, -2.9595696926116943, -2.581754446029663, -5.612542629241943, 8.943078994750977, -9.388174057006836, -4.623846054077148, 9.209752082824707, 1.6276565790176392, -9.036866188049316, -15.546520233154297, -13.92117977142334, 11.17669677734375, -11.39423942565918, -5.04783821105957], [-6.545976638793945, -2.996579647064209, 14.581853866577148, -4.286360263824463, 0.6217989325523376, 12.86359691619873, 14.858097076416016, -0.7987110018730164, 6.263931751251221, 2.464109182357788, -2.5710155963897705, 16.846166610717773, 15.2475004196167, -11.312248229980469, 11.583285331726074, 0.180617094039917, -1.3894826173782349, -6.2747039794921875, 2.212501287460327, -9.323444366455078, -2.944448709487915, 3.937528133392334, -11.297049522399902, 4.367276191711426, 0.24701492488384247, 0.21290676295757294, -0.6304925680160522, 7.37504768371582, -4.168588161468506, -3.580828905105591, -6.472272872924805, 2.5731143951416016, -8.782915115356445], [-2.649296283721924, -3.019362688064575, -2.607560634613037, -0.16985231637954712, 3.710613250732422, 1.7816298007965088, 1.960529088973999, -17.587371826171875, 2.2166402339935303, 3.0783438682556152, 7.532070636749268, 5.241638660430908, -2.8748137950897217, 15.200556755065918, 4.364603042602539, -9.920011520385742, -9.360706329345703, 2.8064005374908447, -2.5583813190460205, -1.212168574333191, 1.757336974143982, 3.209360122680664, 15.373106002807617, 4.442742824554443, 6.020473957061768, -25.178565979003906, 0.9766835570335388, 6.43658971786499, -2.2256033420562744, -2.988685131072998, 7.5934906005859375, 11.027606964111328, 2.703726053237915], [-2.615823745727539, -2.4704036712646484, 17.097871780395508, 4.743019104003906, 17.050060272216797, 3.4043140411376953, 5.773575305938721, 0.08621837943792343, 9.778383255004883, 11.386286735534668, 12.722105979919434, -1.0057669878005981, -5.969446659088135, -11.85556411743164, 1.1578741073608398, 3.615995407104492, 9.082260131835938, -5.274790287017822, -13.891231536865234, 0.6522934436798096, -13.560903549194336, -11.17353343963623, -2.794905185699463, -2.240856409072876, -19.29276466369629, -1.459812045097351, -0.7460664510726929, 9.297740936279297, -0.12957200407981873, -4.157627582550049, 3.329216241836548, -0.9125338196754456, 1.449904203414917], [3.9130313396453857, 4.378889560699463, -1.1826986074447632, -5.742026329040527, 10.853012084960938, 11.524279594421387, -15.971456527709961, 5.319809913635254, 9.556995391845703, -0.1945631355047226, 8.717318534851074, 6.599173545837402, -1.105638027191162, -14.279187202453613, -9.34654426574707, -8.109457969665527, 4.57532262802124, -4.43324089050293, 6.477848529815674, -8.09012222290039, -3.2183117866516113, -3.6871674060821533, -3.79117488861084, -5.718900680541992, 4.594848155975342, -4.066934108734131, 14.49955940246582, -5.592308521270752, 7.214644432067871, 4.928404808044434, -6.531585216522217, 6.636920928955078, -0.8254510760307312], [5.1534528732299805, 25.462846755981445, 19.588611602783203, -5.887712001800537, -5.064395904541016, 9.298622131347656, -5.959225177764893, 14.907772064208984, 3.539153814315796, 13.154743194580078, -18.543699264526367, 5.187285423278809, -19.7698974609375, -0.0915297418832779, -9.445578575134277, 1.0680183172225952, -8.68479061126709, 11.563187599182129, -0.34805724024772644, 6.162438869476318, -4.203060150146484, 17.489776611328125, 11.102877616882324, 10.980149269104004, -3.312865734100342, -9.443535804748535, 5.5666327476501465, -1.1291910409927368, 0.8422598242759705, -10.106501579284668, -25.347326278686523, -3.699399471282959, 7.481156826019287], [1.3029536008834839, 4.65185546875, 6.902068138122559, 6.256063461303711, 8.907791137695312, -4.376938343048096, 5.650646686553955, 13.824979782104492, -2.656768321990967, -0.27983105182647705, -12.43293285369873, 1.7689937353134155, 2.890319585800171, -5.280596733093262, -12.653544425964355, -7.53114652633667, -9.090898513793945, -0.34584760665893555, 9.313055038452148, 3.0047965049743652, 9.883034706115723, 5.883821487426758, 2.4466657638549805, 3.65708589553833, -2.4245119094848633, -3.8049588203430176, 4.854485034942627, 9.25267219543457, -5.070245265960693, 5.92854118347168, 6.103473663330078, 1.9426463842391968, -0.8705125451087952], [5.948312759399414, -22.332054138183594, 12.147027015686035, 9.70286750793457, -2.1192660331726074, 8.290522575378418, 14.801581382751465, -12.21961784362793, -0.13688747584819794, 6.182528972625732, 7.3158159255981445, -8.869730949401855, 8.137887954711914, -24.8936767578125, -1.7879080772399902, -3.379162311553955, 8.357089042663574, 7.771533012390137, -12.798233032226562, 6.901247978210449, 5.9138503074646, -9.506834030151367, -15.277585983276367, -12.527478218078613, -9.98055362701416, 7.287332534790039, -0.9011806845664978, -1.2059519290924072, -1.238924503326416, -8.17336368560791, 12.05926513671875, 8.015141487121582, 9.161288261413574], [-5.218003273010254, -5.577263355255127, -7.923579692840576, 10.006664276123047, -11.1614351272583, -10.448433876037598, 2.701611042022705, 2.682046890258789, -4.401921272277832, -0.6110953092575073, 6.890649318695068, 11.480982780456543, -2.8762996196746826, -2.9570236206054688, 15.508959770202637, -1.5744249820709229, 1.0220005512237549, 7.291465759277344, 0.42867350578308105, 2.507840871810913, -20.449466705322266, 13.57511043548584, -14.040536880493164, -1.0057435035705566, 0.0793163925409317, -1.2271430492401123, 5.6420392990112305, -4.374913215637207, 16.411802291870117, -1.8503538370132446, -2.2175135612487793, 3.570211172103882, -3.1490142345428467], [0.06487063318490982, -5.356050968170166, -3.2260255813598633, -9.626298904418945, 6.323320388793945, -2.4800000190734863, 3.9740772247314453, -1.6277055740356445, 4.552435398101807, -3.1945621967315674, 0.779130756855011, 3.643174171447754, -4.125694751739502, -7.632874011993408, 19.361867904663086, 0.8247717022895813, 0.763947069644928, -17.55929183959961, -8.601597785949707, 9.437830924987793, -1.7485092878341675, -0.17876271903514862, 2.0266900062561035, 2.2228338718414307, 2.993469476699829, -24.48468780517578, 15.205589294433594, -8.826702117919922, 10.208027839660645, 9.175680160522461, -5.245791912078857, -3.354840040206909, 5.582691669464111], [5.9778151512146, 14.647173881530762, -1.2214356660842896, -20.48468589782715, -11.270378112792969, 3.2698171138763428, -4.3453850746154785, 16.43043327331543, -0.05215604603290558, 0.21003934741020203, -4.693493843078613, -1.9735980033874512, -0.7720794081687927, 11.666933059692383, 0.21744436025619507, -18.436386108398438, -12.766692161560059, 18.201065063476562, 8.734142303466797, 3.6976566314697266, 4.963766098022461, -3.5855228900909424, 22.319982528686523, 13.50134563446045, -0.8756234049797058, -4.388492107391357, 0.233103409409523, 1.7982269525527954, -3.5482606887817383, -4.8429179191589355, -17.816875457763672, -5.370988845825195, -6.302536487579346], [1.773871898651123, -0.24605925381183624, 0.47369638085365295, 6.1046833992004395, -3.5888171195983887, -2.9205029010772705, -1.7023704051971436, 2.548740863800049, -1.8910589218139648, 6.055124759674072, -11.153594970703125, -1.01815927028656, 5.135676383972168, 17.502038955688477, -4.928154468536377, -8.270798683166504, -3.0377466678619385, 6.088412761688232, 1.744539499282837, -9.345823287963867, 10.526691436767578, 3.073019027709961, 4.920219898223877, 4.84865140914917, 8.004420280456543, -4.654972553253174, 5.256371021270752, 8.932825088500977, -4.293984413146973, -17.902740478515625, 6.731346130371094, -1.3327771425247192, 7.921527862548828], [-0.5352386832237244, -4.397686004638672, 7.12501859664917, -1.6822631359100342, -2.5500199794769287, -7.29605770111084, 1.9149532318115234, 11.95178508758545, -3.871915340423584, -2.3756744861602783, 2.698122978210449, -2.5693717002868652, -4.8459272384643555, -1.7317699193954468, 0.48192164301872253, 2.7120141983032227, -6.926088809967041, 10.059564590454102, -6.068988800048828, -2.3386855125427246, 1.6743865013122559, -8.930773735046387, 4.088404655456543, -11.468032836914062, 1.336014747619629, 4.375777244567871, -5.5796027183532715, -0.37201374769210815, 2.655548334121704, 6.360853672027588, -4.965774059295654, 18.23995590209961, -3.903888463973999], [-8.215808868408203, 3.6323955059051514, -8.273037910461426, 4.9276933670043945, 2.28883695602417, -3.335695505142212, -11.043691635131836, -2.636821985244751, -3.159848690032959, -3.145740509033203, -17.961339950561523, -2.2249908447265625, 1.5578995943069458, 12.835495948791504, 5.135519981384277, 2.197277069091797, 12.756351470947266, 5.1657304763793945, -1.44271719455719, -0.13698019087314606, -3.4319651126861572, 6.942418575286865, -5.7171831130981445, 15.736202239990234, 10.292829513549805, 4.779109477996826, -15.388223648071289, 3.037229061126709, 8.255278587341309, 6.2176103591918945, 5.6952104568481445, -9.354195594787598, 8.321460723876953], [0.780095636844635, 5.6680707931518555, -5.497167110443115, -2.797954797744751, -5.987177848815918, -2.754312038421631, -3.9635348320007324, 10.318499565124512, 5.7800445556640625, 0.65474534034729, 14.098980903625488, -8.859692573547363, 6.013153076171875, -5.263896465301514, -6.961275577545166, -3.64621901512146, 2.6897196769714355, 1.5002137422561646, -12.489602088928223, -6.0442657470703125, 9.574254035949707, -5.547355651855469, 9.633940696716309, -4.880066394805908, -6.8076558113098145, 0.7783641219139099, 4.614952564239502, -1.9944453239440918, 3.5214104652404785, -7.235286235809326, 1.7277660369873047, 3.227280616760254, -0.054322872310876846]], [[5.12926721572876, 6.12025260925293, -9.661309242248535, -6.524796962738037, 2.07426381111145, -20.57474708557129, -0.14498375356197357, 14.85940170288086, -5.81830358505249, -7.567668914794922, 18.139774322509766, 4.084046363830566, -8.188886642456055, -2.8577699661254883, 3.9437358379364014, -5.606468200683594, -4.996579170227051, -5.188642501831055, 0.8165846467018127, 5.116866111755371, -2.1668689250946045, -0.19844383001327515, 0.21486449241638184, 8.009076118469238, -9.374177932739258, -4.503878116607666, 5.039486885070801, 0.325006902217865, -3.9202921390533447, -3.147010564804077, 9.978720664978027, -2.9545607566833496, -6.358414173126221], [-3.0465807914733887, 2.803076982498169, -3.4125964641571045, -10.32287883758545, -8.094305038452148, -6.468474864959717, 3.863846778869629, 0.7909467816352844, -16.739595413208008, -13.350707054138184, 7.579593181610107, -8.637808799743652, 0.04789530858397484, 0.12917162477970123, 6.599251747131348, 8.649495124816895, -8.569687843322754, -12.834564208984375, -19.30582618713379, -7.951775074005127, -11.487058639526367, 7.099081516265869, 2.4212634563446045, 5.702033996582031, -14.237165451049805, -4.384561538696289, 4.2310471534729, 2.161433219909668, 10.475150108337402, -1.8277080059051514, -0.4924331307411194, 7.512088775634766, -5.618362903594971], [-5.41281795501709, 9.008788108825684, -1.2880878448486328, -0.40322768688201904, -4.626369953155518, -0.9415823221206665, -9.208287239074707, -21.690317153930664, -18.263736724853516, -1.1446082592010498, -7.231328010559082, 8.485099792480469, -6.215226650238037, 6.9818434715271, 1.5359344482421875, 11.879467010498047, 4.22928524017334, 1.4151984453201294, -0.7668256759643555, 5.095019817352295, -3.809152126312256, -3.6867518424987793, 0.14965634047985077, -1.160537838935852, -4.029122352600098, 5.951218128204346, 2.4501142501831055, -9.598516464233398, -11.424602508544922, 7.6300740242004395, -2.9992668628692627, 2.454517126083374, 6.4750285148620605], [-0.1892053484916687, 7.313026428222656, 8.834590911865234, -1.2059993743896484, -1.3065115213394165, 0.5775347352027893, 5.50870943069458, -0.37762993574142456, 3.0905051231384277, 4.998772144317627, 3.0472986698150635, 0.13781142234802246, -4.094387054443359, 1.6172080039978027, -0.3850693702697754, -0.08076924085617065, 15.537672996520996, 8.002060890197754, -13.934042930603027, 0.8749632239341736, -3.5335581302642822, 0.21459323167800903, 1.1096872091293335, -5.11039924621582, -3.812896966934204, 0.10782606899738312, -1.8260200023651123, -1.3732984066009521, -3.606114625930786, -1.3048425912857056, -1.7964987754821777, -2.5344502925872803, 3.5155892372131348], [3.778186559677124, -3.9699692726135254, 2.7141900062561035, 3.4970107078552246, -8.343683242797852, 6.288884162902832, -2.3371949195861816, 0.8843569755554199, 6.842920780181885, -0.9161431193351746, -13.94876480102539, -12.587153434753418, -7.844037055969238, 3.559998035430908, -6.170286655426025, 3.129909038543701, 2.0116312503814697, 11.932465553283691, -2.9838449954986572, 10.375272750854492, 0.12234421819448471, 3.012765645980835, -5.360663414001465, -3.381768226623535, 3.0594475269317627, 13.063859939575195, 3.2638485431671143, 4.369745254516602, -3.880753517150879, 8.792993545532227, -1.2287055253982544, -1.7478004693984985, -3.657799243927002], [-2.7830934524536133, 10.30870246887207, -0.3709850609302521, -1.8000123500823975, 2.7978930473327637, -4.305460453033447, -1.513345718383789, 2.199794292449951, -3.2586781978607178, 12.20891284942627, -13.271886825561523, -17.67734718322754, 3.7625958919525146, -8.93869686126709, 7.007646560668945, 1.9222115278244019, 6.033559799194336, -7.712042808532715, -10.074417114257812, -3.074190855026245, 2.5095958709716797, 5.21204137802124, -6.800446510314941, 4.520770072937012, -4.435696601867676, 2.4734320640563965, -5.77786922454834, 5.4812164306640625, 12.556323051452637, 12.104458808898926, -15.131446838378906, 9.81689453125, 5.95944881439209], [-6.997555732727051, 0.7923486232757568, -0.4945032298564911, 0.8750925660133362, -9.653929710388184, 7.584991931915283, -12.124332427978516, -2.7842023372650146, 5.523523807525635, 2.4180092811584473, -19.285232543945312, 8.404963493347168, -1.8409513235092163, 0.37587031722068787, -10.491334915161133, -4.319388389587402, -0.23734834790229797, -6.62716817855835, -0.23037748038768768, 1.152995228767395, 7.5138325691223145, -13.61303997039795, 19.662172317504883, -2.566429376602173, 17.55185890197754, 4.677890300750732, -4.436403274536133, -8.140388488769531, -4.8030314445495605, -4.4885382652282715, 4.384890556335449, 1.4989858865737915, -1.0200990438461304], [-3.4911861419677734, 10.863211631774902, 1.4045933485031128, -6.694333553314209, -0.7513195276260376, -2.6289186477661133, 11.064131736755371, 3.061356544494629, -1.3910017013549805, 11.548402786254883, -6.005366325378418, -6.496280193328857, 3.6601037979125977, 6.329095840454102, 10.61108112335205, 1.3502699136734009, 6.161664009094238, 2.6375036239624023, -11.598496437072754, 8.054168701171875, -6.714657306671143, 8.724754333496094, -3.681303024291992, -10.154998779296875, -6.127346992492676, -2.215355157852173, 5.494252681732178, 2.016932964324951, -0.24227149784564972, 6.839787483215332, -2.474506378173828, 8.663259506225586, -3.1151418685913086], [-3.735339879989624, 3.0963034629821777, 0.7480475902557373, 3.7411272525787354, 0.17485526204109192, 2.8874495029449463, 3.0324740409851074, 8.168712615966797, -2.323594093322754, 5.742232799530029, 1.2274162769317627, 1.1143275499343872, 5.486639022827148, 2.3258419036865234, 9.267187118530273, -6.6692118644714355, -0.9282635450363159, 5.700461387634277, 4.042328357696533, -0.5582666397094727, 1.2030013799667358, 3.6933882236480713, -12.012757301330566, 7.414885997772217, -6.344390392303467, -12.810466766357422, -12.187833786010742, 13.601914405822754, 1.5122162103652954, 7.166538715362549, -11.746854782104492, -4.338708877563477, 3.6405577659606934], [-1.3516792058944702, -2.149355173110962, 1.688031792640686, -7.633859634399414, -1.0559502840042114, -1.4948307275772095, 6.484303951263428, 0.6945846080780029, 17.9649715423584, -3.180598497390747, 14.133399963378906, 1.408096432685852, 1.0472720861434937, -10.645872116088867, -6.439938545227051, -3.110718011856079, 1.388032078742981, 1.7421748638153076, 0.5848560929298401, -0.8634527921676636, 6.993882179260254, -3.9837090969085693, 13.842300415039062, 8.925442695617676, -3.3933725357055664, -7.911886692047119, 0.9453995823860168, 3.0145678520202637, 7.8250908851623535, 1.173485517501831, -4.835363864898682, 9.50258731842041, -8.657896995544434], [3.895814895629883, 9.358262062072754, 4.357681751251221, -5.382978439331055, 1.1030046939849854, -6.924704074859619, 0.4781103730201721, -10.410197257995605, -4.988284111022949, 3.7097959518432617, -2.7653274536132812, 20.944889068603516, 2.230463743209839, 13.979241371154785, -6.229573726654053, 2.5932869911193848, 19.093584060668945, 6.377896308898926, -4.8932271003723145, 9.500207901000977, 11.279895782470703, 0.0749887153506279, 12.935133934020996, 1.0385254621505737, -9.298535346984863, 4.895534038543701, 11.171608924865723, -2.5301058292388916, -15.526155471801758, -5.884621620178223, 1.3703598976135254, 9.450182914733887, -17.271482467651367], [-4.044345378875732, -3.3507702350616455, 1.6592354774475098, -0.130742147564888, 3.7281086444854736, 1.9231780767440796, 0.2934086322784424, -4.141939640045166, 8.335611343383789, 5.92753791809082, 4.922050476074219, -0.48461583256721497, 3.885194778442383, -8.754446983337402, 4.700279235839844, 5.268160343170166, -7.173266887664795, 15.987272262573242, 9.413734436035156, 4.762573719024658, -0.7866306900978088, -7.448897361755371, -0.8242629170417786, -4.845598220825195, -0.8757588267326355, -1.0863254070281982, 1.5946544408798218, 4.343113899230957, -9.002570152282715, 10.484321594238281, -7.2209062576293945, -9.522137641906738, 4.436868190765381], [7.342027187347412, -0.4089048504829407, -4.126095771789551, 4.020612716674805, 1.3745379447937012, 4.592254161834717, 1.1266285181045532, -4.371282577514648, 0.7155203819274902, 3.764582395553589, 20.175682067871094, 2.01063871383667, -1.4864901304244995, -3.095412492752075, -7.647497653961182, -1.6754531860351562, 8.02756118774414, -1.4186874628067017, -3.0766184329986572, 4.629272937774658, 3.8051886558532715, -1.7604584693908691, 14.636006355285645, 7.02357816696167, 13.860156059265137, -4.151841163635254, 5.055912971496582, 0.5712292790412903, 4.192906379699707, -9.331472396850586, 12.50403881072998, 10.477497100830078, 2.5639727115631104], [-0.4063783288002014, 7.031388282775879, -3.5588958263397217, -11.847833633422852, 3.5130457878112793, -4.6397833824157715, 6.662750720977783, -10.24091625213623, 9.262779235839844, 13.130500793457031, 13.110466957092285, -6.223710536956787, -3.3891735076904297, -20.367265701293945, 6.798608779907227, 1.3832310438156128, -7.780330181121826, 2.9464094638824463, 1.9685274362564087, -4.081309795379639, 7.551967144012451, -0.40648531913757324, 3.418694257736206, 12.011430740356445, -6.2388176918029785, -6.106105804443359, -2.4341979026794434, 4.231339454650879, 8.394957542419434, 15.301913261413574, -0.15854744613170624, 12.822033882141113, -7.746001243591309], [5.157731533050537, 8.275250434875488, 1.2171950340270996, 6.243617057800293, -1.66484797000885, -1.1839109659194946, 0.07867863774299622, 6.795766353607178, -10.722314834594727, -0.14736436307430267, 8.779833793640137, -11.647689819335938, 1.0109378099441528, 5.572220325469971, -3.307912826538086, 6.221316814422607, 6.421782970428467, -9.455008506774902, -7.448299884796143, 2.577374219894409, -6.487456798553467, 2.2962710857391357, -4.960725784301758, 6.184368133544922, 3.7778306007385254, -0.9780175685882568, -0.2928721606731415, -7.088593482971191, 16.28668212890625, -6.512482643127441, -2.409339666366577, -6.251475811004639, -2.2630715370178223], [-5.168306827545166, 10.717559814453125, 6.179570198059082, -1.821410059928894, 5.858489990234375, 3.3796226978302, 8.701751708984375, 2.7362444400787354, -13.894981384277344, 6.957605361938477, -1.4681925773620605, 3.395763397216797, -8.233072280883789, -3.697631597518921, -8.343269348144531, -5.755157470703125, 8.37443733215332, 3.100630283355713, -10.128662109375, -4.662109375, 7.051626205444336, 15.04045581817627, -2.6374502182006836, -2.5530757904052734, 1.2833796739578247, -0.04994427412748337, 8.427483558654785, 4.461216449737549, -4.299308776855469, -0.8802973628044128, -8.35030746459961, -3.171243190765381, -7.643099308013916], [-5.927282810211182, 8.28834342956543, -1.2449390888214111, 5.0671796798706055, -4.883750915527344, 6.955663204193115, 3.905810832977295, 4.406729698181152, -3.0067152976989746, 7.996151924133301, 2.93290638923645, 9.864439964294434, -11.985692024230957, 0.7745233774185181, -14.01978874206543, 0.21451222896575928, 21.939285278320312, -16.806621551513672, -1.376299500465393, 1.017444372177124, -3.718334436416626, 14.704517364501953, -0.636429488658905, 6.708836555480957, 7.386221885681152, -2.4757583141326904, 2.2008800506591797, 4.133163928985596, 7.869063377380371, -4.915943622589111, -4.282524108886719, 10.297477722167969, 11.053019523620605], [-1.6796505451202393, 2.395385503768921, 3.2503583431243896, -4.793112277984619, 1.364269495010376, -8.901863098144531, 3.7005085945129395, -4.5535149574279785, -3.7033615112304688, -5.060251235961914, 2.330810785293579, -3.2334678173065186, 2.4665346145629883, -6.368608474731445, -2.9115147590637207, 3.973639726638794, 6.070474624633789, -5.2412028312683105, -11.81067943572998, -3.5289177894592285, -2.010603666305542, -1.7783470153808594, -5.275247097015381, 2.937055826187134, 4.209906578063965, 3.158742666244507, 4.28208589553833, 9.791549682617188, 0.5648248195648193, 3.5085620880126953, -7.495864391326904, -3.264167308807373, -3.1843085289001465], [-2.7208175659179688, 1.5058014392852783, -9.246733665466309, -3.2530441284179688, -4.758816719055176, 4.143908977508545, -2.9491755962371826, 4.766934871673584, -1.2312434911727905, 10.752772331237793, 1.3445096015930176, -2.9976820945739746, -12.879830360412598, -5.067811489105225, 2.809141159057617, -2.7467336654663086, -7.806192398071289, -1.5966721773147583, -0.394413024187088, -2.0299670696258545, 0.07400505244731903, 2.588561773300171, -7.000499248504639, -4.096899509429932, 2.126852512359619, 3.2184360027313232, 4.669248104095459, 1.8268945217132568, 6.51551628112793, 1.8668171167373657, 10.437756538391113, 7.263126373291016, 2.9061524868011475], [0.5261673331260681, 16.812402725219727, 4.910192012786865, 1.6711472272872925, 0.059654347598552704, 2.3074262142181396, -3.560762882232666, 14.289046287536621, -5.426193714141846, -10.569576263427734, -4.465760231018066, 5.555513381958008, 7.042176246643066, 11.858427047729492, -10.313467025756836, 6.373095989227295, -0.06314206123352051, -7.482369422912598, -1.9467579126358032, 2.8148508071899414, -6.882583141326904, -10.877751350402832, 5.286487579345703, 2.7697503566741943, 2.4798436164855957, -0.07535825669765472, -15.33836841583252, -10.355694770812988, 5.223166465759277, -2.6592376232147217, 4.858939170837402, -22.74517059326172, 5.79780387878418], [-2.688455581665039, -1.3227547407150269, 4.038397789001465, 2.9607174396514893, 6.2073283195495605, 14.52239990234375, -2.5700082778930664, 2.570627450942993, -5.578660011291504, -6.609757900238037, -2.8573520183563232, -9.862476348876953, -2.8726110458374023, 1.6574006080627441, 2.1793181896209717, -3.7617247104644775, -1.1209189891815186, -3.7053792476654053, 0.1841469556093216, -6.298988342285156, -3.4015719890594482, 3.93251895904541, -0.6925767064094543, -0.55546635389328, -1.3535436391830444, -14.938965797424316, -5.870887279510498, 2.0390970706939697, 10.71086597442627, 0.4990788996219635, -7.475604057312012, 9.861435890197754, -10.478028297424316], [-9.607545852661133, -1.7981295585632324, 0.7262812256813049, -2.47208833694458, 2.0042858123779297, -1.1945298910140991, -7.696051597595215, 4.027542591094971, 0.6672344207763672, -2.0520808696746826, -7.262204170227051, 16.314340591430664, -0.6027905344963074, -10.796324729919434, -5.606436729431152, -2.1613850593566895, 5.515434265136719, -8.468683242797852, 3.1842997074127197, -2.8018264770507812, 4.948037624359131, -4.170745372772217, -2.0430665016174316, -4.9898529052734375, -10.838275909423828, -0.008421530947089195, -3.171238899230957, 3.349121332168579, -1.3020164966583252, 7.798864364624023, 2.1661195755004883, -6.020489692687988, -1.7884247303009033], [-3.885554075241089, 2.836010217666626, 5.549925327301025, 12.751692771911621, -0.8464283347129822, -1.5129363536834717, 5.5970635414123535, -1.389519214630127, 2.4841315746307373, 10.575284004211426, -4.270429611206055, 1.0284650325775146, -6.463495254516602, 0.3076123297214508, 5.785319805145264, -8.421876907348633, 9.868132591247559, 11.02857494354248, 8.677536010742188, -4.38594913482666, 25.79981803894043, -9.820182800292969, 1.7724603414535522, 4.534811019897461, -8.468568801879883, 1.92605459690094, -7.458647727966309, 4.15858793258667, 3.901348114013672, 4.766311168670654, -4.9986958503723145, 2.829692840576172, -4.533857345581055], [6.371703147888184, 2.4575955867767334, 8.608536720275879, -5.355981349945068, 10.393101692199707, 5.800359725952148, 3.176593780517578, 1.9304815530776978, 0.10986641049385071, -3.992187261581421, 14.22744369506836, -4.051436901092529, -4.233600616455078, -1.3129616975784302, -0.09625911712646484, -6.370362758636475, 1.4879791736602783, 3.6956000328063965, 8.345229148864746, -6.929062366485596, -5.802518367767334, 3.610461950302124, 0.5934748649597168, 0.3556887209415436, -2.9667179584503174, -6.002821922302246, -4.453812122344971, -8.929425239562988, -2.5006489753723145, 2.3117728233337402, 4.343348503112793, -9.323534965515137, -4.821600914001465], [-3.4439361095428467, 1.2999824285507202, 2.3061318397521973, 0.16839684545993805, 0.4443807303905487, -0.5945166349411011, -1.131567120552063, 16.73334312438965, -4.075651168823242, -3.6428980827331543, 3.637099266052246, -6.62680196762085, -1.887921929359436, 10.195590019226074, -2.940761089324951, -0.06656665354967117, -0.28077998757362366, -5.9717864990234375, -6.901870250701904, -11.379749298095703, -4.893462657928467, -0.6309843063354492, -2.7794899940490723, 5.2290496826171875, -8.339165687561035, 10.981006622314453, 1.1621801853179932, -2.343853712081909, -7.331269264221191, 0.4718489944934845, 3.1845309734344482, -3.3970742225646973, 3.675616502761841], [-3.3831183910369873, 0.6000730991363525, -0.7616589665412903, 4.886730670928955, 7.158287048339844, 10.990705490112305, 3.3586843013763428, 2.0099174976348877, 1.3330893516540527, 4.853514671325684, 1.5789148807525635, 4.2790985107421875, 9.466925621032715, 2.076836109161377, 5.215747833251953, 3.128114700317383, -6.208014488220215, 6.921061038970947, 2.9769318103790283, -3.7451634407043457, -5.642176628112793, -7.942002773284912, 5.052419662475586, -7.8346781730651855, 0.6376706957817078, -6.366624355316162, -2.6779720783233643, 2.3898417949676514, 5.195304870605469, 2.216521739959717, 6.3150458335876465, -9.036489486694336, 6.348038673400879], [-0.021234171465039253, 7.002891540527344, -0.16911117732524872, 4.532028675079346, 0.18668147921562195, 6.8851213455200195, 6.475738048553467, -3.9946351051330566, -11.277442932128906, -3.4795076847076416, -2.061457395553589, -0.026552315801382065, 13.389542579650879, 13.200706481933594, 5.694863796234131, 1.8582496643066406, -2.8543221950531006, -9.525845527648926, -3.2896199226379395, -7.420522689819336, -2.8282558917999268, -2.038006544113159, 3.533623695373535, -5.268431663513184, -3.1596856117248535, -6.439690589904785, 0.7579070329666138, -2.681208848953247, 22.332597732543945, -5.459152698516846, 1.75040602684021, -3.228213310241699, -5.853449821472168], [-9.537341117858887, 14.444742202758789, 7.446135997772217, -1.7452303171157837, -7.1448893547058105, -2.157086133956909, -2.7972636222839355, -12.142982482910156, 13.71038818359375, -7.006120681762695, -6.18690824508667, -0.13134105503559113, 0.2808321416378021, 1.405973196029663, -1.5128499269485474, 7.278958797454834, 2.0838539600372314, 3.245858669281006, 3.1045632362365723, -0.22364524006843567, 6.0551300048828125, -19.231082916259766, 11.113965034484863, -0.10215733200311661, -1.5492005348205566, 0.37131911516189575, -13.344029426574707, -3.6845571994781494, -14.495912551879883, 0.7445057034492493, -2.2891416549682617, -2.89595627784729, 8.37363052368164], [-0.820909321308136, 3.6591432094573975, -3.3542606830596924, -8.53786849975586, -2.3538992404937744, -10.319723129272461, -7.638205528259277, 9.20992660522461, -7.867120265960693, -16.835145950317383, -6.099440097808838, 9.238898277282715, -0.04464638605713844, 3.7080864906311035, 1.9075565338134766, 6.527114391326904, 6.63657808303833, -12.719649314880371, -5.897843360900879, 3.113804578781128, -1.061010479927063, 1.1923998594284058, 3.2697184085845947, -0.996074914932251, 1.0856196880340576, 1.3629848957061768, -2.811030149459839, -8.426819801330566, -7.645430088043213, -2.1642391681671143, -2.1564626693725586, -2.1905715465545654, 0.45463287830352783], [-3.566864490509033, -8.267220497131348, 12.335600852966309, 6.556268215179443, -15.559389114379883, -5.7734375, -1.5703049898147583, -5.309004783630371, 1.1132652759552002, -0.021579762920737267, -1.2857745885849, -5.5884270668029785, -7.851442813873291, 9.94504165649414, -1.5807163715362549, 1.9298619031906128, 7.837584972381592, 2.4516396522521973, -14.275025367736816, 4.248976707458496, 0.07910402864217758, 1.334618330001831, 2.821399688720703, 7.05705451965332, 8.853960990905762, 8.565204620361328, -5.202210903167725, -15.343596458435059, -11.9158296585083, -9.457019805908203, 5.121026039123535, -11.851780891418457, 12.29324722290039], [7.719517230987549, -7.710042476654053, -13.204667091369629, -2.4929442405700684, -1.1173174381256104, 8.638842582702637, -15.697571754455566, 15.031363487243652, -7.292102813720703, -7.942386627197266, 9.134244918823242, -13.540982246398926, -1.5063577890396118, -5.909646987915039, -3.423020839691162, 1.2169225215911865, -22.0296630859375, 3.330240488052368, -4.540587425231934, -2.1168718338012695, 2.2261483669281006, -7.288033962249756, -4.105721950531006, 5.965975284576416, 6.777810096740723, -14.543037414550781, -7.814471244812012, -0.48843517899513245, 12.80488395690918, -5.674083709716797, -9.036001205444336, 14.444400787353516, -1.7334178686141968], [2.0550265312194824, -3.8431711196899414, 3.40311861038208, 6.135362148284912, 3.4757866859436035, -5.774060249328613, 0.7321188449859619, 5.27484655380249, -4.556191444396973, 10.51559066772461, -1.744891881942749, 2.3364193439483643, 3.2096850872039795, 5.39171028137207, 6.145876407623291, -6.377962589263916, 3.571326732635498, 6.822876453399658, 2.6432619094848633, 1.2515751123428345, 2.661284923553467, -9.476049423217773, -18.781068801879883, 9.204005241394043, 0.5268454551696777, -0.8574509620666504, -11.051966667175293, 7.816394805908203, -11.599076271057129, 0.4903213083744049, -9.650680541992188, 5.737936019897461, 13.644125938415527], [-3.231757640838623, -4.424407005310059, 9.149131774902344, -4.929352283477783, -3.9596505165100098, 6.932478427886963, 3.3914108276367188, -8.180048942565918, 1.3307913541793823, -1.4357661008834839, -2.5322422981262207, 2.508920907974243, -3.540233612060547, 3.2864127159118652, -0.8553146719932556, -15.284995079040527, -1.0289644002914429, -1.7912888526916504, -12.608526229858398, -13.475000381469727, 10.836193084716797, -1.2070889472961426, 16.40321159362793, -3.054865837097168, -1.0710084438323975, 2.004441499710083, 7.543216705322266, -0.2085954248905182, -1.5222870111465454, -17.863204956054688, 2.0537025928497314, -0.16446326673030853, -14.741239547729492], [13.592538833618164, -0.45625507831573486, 14.226188659667969, -1.880083441734314, 17.5870418548584, 3.69136905670166, -3.619333028793335, -2.6835761070251465, -7.519534111022949, -1.2739375829696655, 2.23879075050354, -8.252741813659668, 11.703373908996582, -7.707420825958252, -5.597445487976074, 2.40289044380188, 7.876091957092285, 8.437176704406738, -4.7969512939453125, 10.760303497314453, 0.3518054485321045, 7.815166473388672, -8.338354110717773, 2.8131725788116455, -1.5829451084136963, 11.142396926879883, -4.597969055175781, -6.488497734069824, -0.3993634581565857, 16.969295501708984, -7.621570110321045, -2.1712753772735596, -3.62359619140625], [5.002583980560303, 2.8322677612304688, 2.874229669570923, -7.09215784072876, -7.9776787757873535, -8.83061695098877, 2.8191452026367188, -0.4925452768802643, 14.676082611083984, 2.2990357875823975, 7.8234686851501465, 9.773223876953125, -14.04509449005127, -0.7939716577529907, -8.388596534729004, -9.981821060180664, 22.04115867614746, 5.202796459197998, -8.365250587463379, 10.707409858703613, 1.5877270698547363, 8.248231887817383, 3.4140608310699463, 6.101407051086426, -9.5711088180542, 7.578052997589111, 15.520681381225586, 12.188175201416016, -4.538333415985107, -10.691219329833984, -11.479130744934082, 24.586030960083008, -7.229531764984131], [7.02212381362915, 1.4607645273208618, -8.29534912109375, -2.137927770614624, 9.508484840393066, -3.738471746444702, 7.893978595733643, 5.46429967880249, -3.482405424118042, 0.328911691904068, 8.148369789123535, 4.062413215637207, -8.88076400756836, 1.4787168502807617, 4.2011260986328125, 2.710953712463379, 5.95627498626709, -3.374673366546631, 5.12165641784668, -5.636013031005859, -11.647116661071777, 6.7558274269104, -17.513656616210938, -0.603200376033783, -7.347033977508545, -3.1358282566070557, 8.043591499328613, 5.14179801940918, -4.520224571228027, 3.491877555847168, 11.869624137878418, 16.305801391601562, 2.459134340286255], [-1.4976444244384766, 7.508451461791992, 5.839845657348633, -0.5521979928016663, -0.7365710139274597, -11.00364875793457, -8.789041519165039, -2.6488661766052246, 1.2963519096374512, -5.495238304138184, -15.184693336486816, 5.229455947875977, -2.475221633911133, -1.751495361328125, -7.3928141593933105, 6.335696220397949, -0.657189667224884, -2.593374490737915, -1.6913814544677734, -1.2228128910064697, 6.007756233215332, -5.5088887214660645, 7.336070537567139, 6.114424705505371, -0.6114462018013, -0.018662715330719948, 0.1809939295053482, 4.5228047370910645, -19.4488525390625, 5.18173885345459, 6.785139560699463, 4.4463958740234375, 3.7220842838287354], [-8.256945610046387, 9.675324440002441, 1.1460856199264526, -1.7084813117980957, 7.342717170715332, 14.523187637329102, 1.955449104309082, -11.5950288772583, 2.527299404144287, 11.928647994995117, 2.646319627761841, 5.473413467407227, -4.220026016235352, -10.497663497924805, 4.360406398773193, -3.3092174530029297, -1.623796820640564, -0.009242826141417027, 1.5759837627410889, -9.312894821166992, 4.393401622772217, -5.46554708480835, 12.70257568359375, 9.523494720458984, -5.451197147369385, -10.886197090148926, -6.239516735076904, -4.704056739807129, 13.526199340820312, 3.430316209793091, -2.843733549118042, 8.151775360107422, -3.6729817390441895], [6.839698791503906, -5.025627136230469, -1.2444735765457153, 9.683940887451172, 4.790092945098877, 4.636201858520508, -7.445401668548584, -1.0163240432739258, 14.325352668762207, 6.591378688812256, -0.2701645493507385, 10.590923309326172, 1.8942235708236694, -0.9300199747085571, 2.2139790058135986, -4.040566921234131, 11.91983699798584, -6.829459190368652, 14.24714183807373, 5.221562385559082, 0.25481534004211426, -5.98687744140625, 2.810474395751953, -7.690773963928223, 5.9097676277160645, 2.09958553314209, -2.0052313804626465, -20.8591251373291, 0.45711150765419006, -12.746143341064453, 11.814834594726562, -10.039557456970215, 4.2554097175598145], [-0.5252870917320251, -5.080809593200684, -3.390239953994751, -0.9516950845718384, -0.09322688728570938, 10.347620964050293, -3.04704213142395, -0.4728214740753174, -5.780673027038574, -15.319239616394043, 0.745536744594574, 13.334519386291504, 7.427163124084473, 3.7582786083221436, 1.576539158821106, -0.8889127373695374, -10.25909423828125, -1.003078579902649, 1.1069012880325317, 6.384336471557617, -5.282480239868164, 0.5634915828704834, 1.59969961643219, 6.0186333656311035, 8.112887382507324, -10.489070892333984, -10.953981399536133, 4.100901126861572, 7.309915542602539, 3.0451862812042236, -4.77431058883667, -0.4929389953613281, -6.248489856719971], [0.6722157001495361, 0.35368791222572327, -0.1302691549062729, -5.740840911865234, 1.8162659406661987, 3.12744402885437, -0.7622382640838623, 5.744799613952637, -2.1523003578186035, -16.290912628173828, -7.148130416870117, 21.61608123779297, -0.8517904877662659, 1.1035701036453247, -3.273364305496216, -5.681060314178467, -3.4056758880615234, -15.79104232788086, -3.6803383827209473, -5.69221830368042, -1.2008830308914185, -4.084487438201904, 13.928886413574219, -0.1636507511138916, -3.5890064239501953, -0.3503071069717407, -0.22082768380641937, -2.6544535160064697, 6.873106002807617, -11.218293190002441, 3.1424636840820312, -6.577096939086914, -5.06137752532959], [-1.9140857458114624, 2.1588571071624756, -6.956421852111816, 5.341240882873535, -9.38922119140625, -10.847405433654785, -1.483856201171875, -2.6034939289093018, -5.731932163238525, 3.8883426189422607, 2.1151936054229736, -8.812342643737793, 5.769262790679932, -4.7329487800598145, 8.908584594726562, -1.20584237575531, -6.653593063354492, 1.9241716861724854, 3.2664294242858887, -0.09668492525815964, 1.7259366512298584, -9.752056121826172, 7.862462043762207, 3.1623382568359375, 4.518969535827637, 1.7758152484893799, -3.4801297187805176, 1.4555127620697021, -6.069052219390869, -3.6966538429260254, -7.457249164581299, 5.768270969390869, -1.3988442420959473], [8.08789348602295, 20.128503799438477, 0.01882198452949524, -2.859282970428467, 11.4788179397583, -1.7931197881698608, 4.624329090118408, 9.15678882598877, 18.57564926147461, 6.955244064331055, 5.819497585296631, 0.24581804871559143, 17.047422409057617, -13.796147346496582, -1.730672001838684, 0.5881606936454773, -4.83615255355835, 7.504878520965576, 9.170116424560547, 11.438133239746094, 6.798529624938965, -6.241730690002441, -15.891637802124023, 0.3821139633655548, 5.100304126739502, -8.419267654418945, -3.0813632011413574, 15.325162887573242, 7.579590320587158, 7.040639400482178, -14.022380828857422, 14.80817699432373, 6.556828498840332], [-2.36590313911438, 3.492727756500244, -2.4514577388763428, 8.661916732788086, -3.645720958709717, 11.486529350280762, 0.045958857983350754, -13.787806510925293, -6.041306495666504, -3.196556568145752, 1.019171953201294, -2.2748687267303467, -6.752065658569336, 1.91057550907135, 5.457672595977783, -3.890078544616699, -8.570552825927734, 11.886309623718262, 5.36501932144165, 3.0768909454345703, 3.6378519535064697, 5.170735836029053, -6.690108776092529, -15.313063621520996, 1.2488871812820435, 1.2715433835983276, 0.8718510270118713, 1.2269864082336426, 1.294480800628662, -1.0683685541152954, -7.294846534729004, -6.460434436798096, -8.468720436096191], [-1.0877587795257568, -12.275045394897461, 6.051975250244141, -1.9990702867507935, -4.528440952301025, -10.836787223815918, 8.595115661621094, -4.419748306274414, 5.162054061889648, -4.875009536743164, -7.920888423919678, 3.661456346511841, 12.16092586517334, 3.4540040493011475, 4.243572235107422, -2.7937278747558594, 6.0138373374938965, -6.174261093139648, -8.530152320861816, -11.088802337646484, -5.135953903198242, 9.634617805480957, -8.046250343322754, 0.2321215718984604, 0.14160166680812836, -4.023986339569092, 9.892634391784668, 4.371908664703369, 6.332740306854248, -3.423485040664673, -4.467264175415039, -1.2322766780853271, 6.799574851989746], [1.55111825466156, -5.666504859924316, -3.266367197036743, -5.552490711212158, -5.44340181350708, 2.6159145832061768, 7.773230075836182, 0.9990871548652649, 3.8765828609466553, 5.829041004180908, 3.1874125003814697, -0.536426305770874, -5.651592254638672, -4.281179428100586, -16.07438850402832, -5.294283390045166, 2.7137811183929443, -10.05334186553955, 4.533334255218506, 0.19306504726409912, -5.209950923919678, 14.437047958374023, 1.514504075050354, -13.625055313110352, 2.3998615741729736, 8.873819351196289, 19.597105026245117, 3.7143537998199463, 0.3437781035900116, -16.947900772094727, -6.433199405670166, 13.028002738952637, 0.02775455079972744], [16.49308204650879, -7.3845438957214355, -5.795313835144043, 4.097408294677734, 10.360581398010254, -9.28702163696289, -4.771024703979492, 12.41601848602295, -6.817447185516357, -11.61917495727539, -5.119915008544922, -8.686796188354492, 15.16140079498291, 1.4079792499542236, -3.355755567550659, 5.0540289878845215, -6.092559814453125, -1.1729094982147217, -0.24883157014846802, 9.856100082397461, -10.091862678527832, -11.358131408691406, -6.7841715812683105, -4.438281536102295, 13.674695014953613, -5.723862648010254, -0.24316872656345367, -3.297149419784546, -3.9838814735412598, -3.3841161727905273, -1.9078508615493774, 0.7833583950996399, 10.605181694030762], [-0.6466430425643921, 1.6006962060928345, -11.742232322692871, -4.921543598175049, -12.391317367553711, 3.2712020874023438, 7.048840522766113, 15.959430694580078, -21.020002365112305, -5.766855239868164, 7.149899959564209, -3.8101704120635986, -3.8677847385406494, 13.158215522766113, -5.979290962219238, 9.656250953674316, -0.6760522723197937, -6.082196235656738, -20.97351837158203, -2.353301525115967, 0.6750511527061462, 18.32884979248047, 13.758356094360352, 6.2783522605896, -10.2620267868042, -1.3645769357681274, 9.771592140197754, 1.70370614528656, 14.893047332763672, 3.0523767471313477, -8.959176063537598, 14.296344757080078, -11.469686508178711], [2.156076192855835, 4.092724800109863, 4.410905361175537, -8.004810333251953, 7.249868392944336, -9.80551815032959, 1.0147125720977783, 0.17020754516124725, 5.136854648590088, 17.841951370239258, -2.7036023139953613, 15.457837104797363, 5.071734428405762, -7.471396446228027, 5.561765670776367, 3.557783365249634, 3.3252205848693848, 9.245379447937012, 0.7674669623374939, 12.46203899383545, 2.4307162761688232, 2.0355193614959717, -4.610538959503174, 10.382955551147461, -9.308405876159668, -9.046039581298828, 3.576606273651123, 16.49570655822754, -7.873782634735107, 15.628462791442871, 7.294232368469238, 6.59728479385376, 4.511597156524658], [-4.282046794891357, -0.7629349827766418, -0.038782406598329544, -3.114027738571167, -0.26052457094192505, 11.482000350952148, 3.9324467182159424, -6.090694427490234, -0.39743053913116455, 13.177976608276367, 0.4727092981338501, -1.7207876443862915, 4.548304557800293, -9.58448600769043, 0.9463545083999634, -1.6304105520248413, -4.789106845855713, 2.691216230392456, -1.5153114795684814, 3.0668041706085205, 1.0012718439102173, 4.729278564453125, -2.226691246032715, -5.190983772277832, 10.042343139648438, -3.2453126907348633, -0.5716407299041748, -2.7648980617523193, 6.348464488983154, -10.271472930908203, -8.083150863647461, -1.7827236652374268, 1.7326925992965698], [-0.7637659907341003, 4.720623016357422, -9.25903034210205, -9.378430366516113, -1.8972697257995605, 2.7377946376800537, -0.30430683493614197, 1.3296246528625488, -5.6300368309021, -14.584529876708984, -12.10543441772461, 6.3375163078308105, -5.7275614738464355, -0.9076954126358032, -2.210435628890991, 5.01557731628418, -8.463861465454102, -10.100976943969727, -5.21522331237793, 0.20217972993850708, -13.138693809509277, -8.202117919921875, 1.9514671564102173, 5.884544372558594, 10.32982349395752, -7.677826404571533, -2.0299010276794434, -2.836759090423584, -0.8176016807556152, 0.06380408257246017, -1.4938547611236572, 0.6032981872558594, 6.528635501861572], [-6.159926414489746, 5.7162909507751465, 1.394585132598877, -2.911868095397949, 9.682589530944824, -11.819367408752441, -3.9556047916412354, -5.853611946105957, -0.8484841585159302, 1.2744882106781006, 8.06541919708252, 7.409560680389404, 8.511651039123535, -6.378244876861572, 2.9126040935516357, 0.7508435249328613, -2.538048505783081, 0.24375540018081665, 11.27554702758789, -0.5520485639572144, 4.826879024505615, -1.8335059881210327, 7.2506232261657715, 0.21623364090919495, -0.8992792367935181, -14.69296932220459, 2.81481671333313, -7.187169551849365, -12.182639122009277, -2.230914831161499, 13.698100090026855, 11.206790924072266, -11.70993709564209], [0.6560680866241455, -6.537071704864502, 1.5477263927459717, 4.393067836761475, 0.7927525639533997, 7.169142246246338, -9.097475051879883, 1.1376419067382812, 1.830843210220337, -2.220224380493164, 9.185110092163086, -11.201231002807617, -2.2722277641296387, 0.04475640133023262, -8.545550346374512, 4.061976432800293, -3.6611013412475586, -3.972681999206543, -8.567524909973145, -8.510431289672852, -9.255043983459473, -1.650554895401001, -4.978361129760742, 2.954177141189575, -3.5769801139831543, 2.3259809017181396, -1.4207358360290527, 3.605933427810669, 3.194052219390869, 5.392723083496094, -2.9900825023651123, -8.931608200073242, 10.072288513183594], [2.7611544132232666, 6.307938575744629, 0.3824690282344818, 1.567531943321228, 7.422191143035889, 11.330527305603027, 2.553277015686035, -3.4584717750549316, 9.659635543823242, 9.558358192443848, 12.178372383117676, -8.992735862731934, 2.0211002826690674, -3.3634772300720215, 2.525120973587036, -3.2397990226745605, -2.0163514614105225, 14.07142448425293, 13.767129898071289, 0.08107954263687134, 13.969795227050781, 3.075756788253784, 9.459367752075195, -4.323866844177246, -3.7732744216918945, 0.10521858185529709, -4.525468349456787, -11.793564796447754, 8.487994194030762, -3.19077730178833, 3.224958658218384, -1.8472473621368408, -14.481194496154785], [-2.4441025257110596, 9.74899673461914, -2.6538639068603516, -6.666586875915527, 5.365810394287109, 5.080778121948242, 4.168717861175537, 2.7817225456237793, -0.7742964625358582, -2.4734418392181396, 2.5845439434051514, 7.143464088439941, 0.9337916970252991, -5.834920406341553, -5.737117767333984, -1.4889990091323853, -3.9747474193573, -11.03571891784668, 4.243157386779785, -0.25248509645462036, 3.3668651580810547, -0.45913466811180115, -1.755799651145935, 6.373234748840332, -2.516190767288208, -10.005681991577148, -7.80436372756958, 0.2746978998184204, 12.082317352294922, -2.5397462844848633, 3.9407665729522705, -8.528943061828613, -3.4313902854919434], [-0.9543927907943726, -10.067397117614746, 0.780753493309021, 3.961203098297119, -5.597276210784912, -10.643281936645508, 8.489396095275879, -2.88665509223938, -4.5753302574157715, -1.7707923650741577, -12.724868774414062, 3.2168917655944824, 2.8366591930389404, 16.076011657714844, -2.8004486560821533, -10.047508239746094, 8.907301902770996, -10.01409912109375, -11.288005828857422, 2.069474697113037, 1.1363660097122192, -1.652612328529358, 8.827506065368652, -9.956559181213379, 3.1507508754730225, 5.539539337158203, 11.986350059509277, 2.8805456161499023, -0.4946507215499878, -10.353433609008789, 7.411332607269287, 3.827805995941162, -12.772038459777832], [7.510767936706543, 7.2914137840271, 3.0673816204071045, 0.7469556927680969, -4.28940486907959, -3.37105655670166, 2.877518892288208, -3.2046444416046143, 10.454021453857422, -2.6865830421447754, 2.5617566108703613, 2.66143536567688, 3.092664957046509, 1.4476850032806396, -1.7315711975097656, -0.6064587831497192, 11.770336151123047, -5.652862548828125, -4.878750801086426, 4.814155578613281, -1.31123948097229, -0.868001401424408, 12.709814071655273, 7.912712574005127, -0.9876248836517334, -7.0931315422058105, 3.0345499515533447, -5.677997589111328, 12.793347358703613, -2.0760722160339355, -8.79877758026123, 15.266136169433594, -4.244735240936279], [11.81619930267334, -7.276889324188232, -3.883005142211914, 3.17702054977417, 2.763184070587158, 16.411319732666016, -8.06605052947998, 0.38205137848854065, 9.391880989074707, -10.15893268585205, 1.696460247039795, -6.254543781280518, 3.1277167797088623, -3.0447001457214355, -8.739206314086914, -3.353050708770752, 1.1544698476791382, 14.787943840026855, -1.0157887935638428, 8.230270385742188, 7.8042778968811035, 0.4037940800189972, -2.68007230758667, -20.22357749938965, 8.098013877868652, 3.553677797317505, 1.1379292011260986, -4.419875144958496, -3.5039565563201904, -15.344715118408203, -0.0019977507181465626, 1.6890783309936523, -3.4233896732330322], [3.455735683441162, -13.615988731384277, 0.5673679113388062, 1.4726612567901611, -0.571491003036499, -4.217789173126221, 5.018685340881348, -6.309725761413574, -2.6578142642974854, 3.5710079669952393, 1.3199890851974487, 1.3009542226791382, 7.584447383880615, -7.2320990562438965, 8.839600563049316, -5.231574058532715, -11.021760940551758, 13.895377159118652, -2.389491319656372, 14.09706974029541, -1.1099889278411865, 13.759944915771484, -12.812066078186035, 1.9308732748031616, -2.918410062789917, -7.819470405578613, 7.091878890991211, 9.835027694702148, 3.852907419204712, 11.866390228271484, -3.8585336208343506, 11.526182174682617, -8.814577102661133], [0.1423354148864746, -5.429145812988281, 1.951238989830017, 3.0416579246520996, 5.176758289337158, -2.471205711364746, 5.64329195022583, 2.85874342918396, -1.310019612312317, 9.937952041625977, -1.4488513469696045, -6.751785755157471, 2.6468639373779297, 2.189736843109131, -2.4293854236602783, 12.275403022766113, 11.596651077270508, 1.0837762355804443, -7.399302005767822, 4.068305492401123, -10.313374519348145, 1.0670356750488281, -4.9493021965026855, -7.23646879196167, 9.708979606628418, 11.818069458007812, 4.8523101806640625, -0.9691553711891174, -5.00072717666626, 4.379668235778809, 9.98491096496582, -4.241545677185059, 11.39661979675293], [-0.02950255014002323, -6.966255187988281, -1.6224130392074585, -2.318392276763916, 5.428876876831055, 0.32650846242904663, -7.266077518463135, 4.1377153396606445, 2.7192070484161377, 3.6187756061553955, 2.972438335418701, 13.067523002624512, -1.7815531492233276, -0.4448409974575043, -4.454094409942627, 2.682154893875122, -6.113603591918945, 7.782494068145752, -2.830329418182373, 18.152080535888672, -1.7881261110305786, 4.671609878540039, 0.12041735649108887, 3.0050201416015625, 3.0624117851257324, 0.47070959210395813, 6.527609348297119, 0.5321120619773865, -2.873551368713379, 5.685860633850098, 8.088774681091309, 1.9065380096435547, -4.168586730957031], [-1.0957802534103394, 0.5650029182434082, 1.1089327335357666, -0.675561249256134, 1.5157880783081055, -8.174392700195312, 9.960348129272461, 4.278016567230225, 2.3809783458709717, -7.787114143371582, 2.1020801067352295, 6.19543981552124, 4.040562152862549, 3.04876446723938, -6.50686502456665, -2.46927809715271, -1.7359817028045654, -4.005986213684082, 0.9737915396690369, -2.685668706893921, 1.541602373123169, -6.496561527252197, -4.082827568054199, 9.99370288848877, -6.842739105224609, -2.7209811210632324, -3.057215452194214, 15.644082069396973, -4.842922687530518, 2.2666096687316895, 0.24413847923278809, 7.811969757080078, 5.805978298187256], [4.529181957244873, 4.551257133483887, -9.078509330749512, 8.15837574005127, 12.427392959594727, -0.1851847916841507, -6.839411735534668, 8.705320358276367, 1.3661115169525146, -0.06116652861237526, 9.132012367248535, -15.153067588806152, 13.237977027893066, -5.689805030822754, 10.429425239562988, -5.797090530395508, -14.970955848693848, 15.45935344696045, 9.428157806396484, -4.605119705200195, 12.104040145874023, 6.585611343383789, -9.403092384338379, 1.3943445682525635, -11.760294914245605, -13.693916320800781, -4.466300964355469, -1.8338414430618286, 2.602057695388794, -0.05144743248820305, 2.4691600799560547, 10.295051574707031, -5.523767471313477], [-13.531705856323242, -2.5494706630706787, -3.384936571121216, -4.809021949768066, -2.921056032180786, -0.8240559101104736, 8.253859519958496, -5.19989538192749, -1.318398356437683, -7.658502578735352, 16.518869400024414, 5.83347225189209, 0.158625528216362, -5.772946357727051, 2.0724806785583496, 3.468961000442505, -13.089715003967285, 10.558859825134277, -8.474401473999023, -6.337454319000244, -2.6090524196624756, -6.081888675689697, 1.587207317352295, 13.201602935791016, -9.93140697479248, -9.754987716674805, 3.338740825653076, 13.231590270996094, 4.663857460021973, 2.558411121368408, -4.230323791503906, 1.246241569519043, -0.14198821783065796], [-3.634838819503784, 2.949030876159668, -5.134086608886719, -4.109097480773926, 0.056634411215782166, -1.7335118055343628, -5.935024738311768, -4.385621547698975, 7.261783599853516, -5.631328105926514, -2.929664373397827, 14.216068267822266, 2.211611270904541, -11.3057861328125, -8.304064750671387, 1.4569361209869385, 3.4504098892211914, -10.500232696533203, -1.4613009691238403, -6.728175640106201, -3.010443925857544, -5.919472694396973, 11.794950485229492, 2.343576669692993, 3.380321979522705, 3.7874786853790283, -0.49608859419822693, -3.5430355072021484, 1.0857206583023071, -11.510805130004883, -6.420194149017334, 7.681434631347656, 7.979320049285889], [2.305417537689209, 7.851565361022949, -3.9188177585601807, 0.11625774204730988, 0.04505516588687897, 5.551936626434326, -3.7551612854003906, 0.5366564989089966, 7.873528480529785, 3.712677240371704, 3.1944682598114014, 6.381148815155029, -8.551352500915527, -2.203643798828125, -12.231073379516602, 6.2576584815979, 5.154158115386963, -7.118194580078125, 8.298248291015625, -5.388333797454834, -2.3270199298858643, 1.7958427667617798, -5.596639633178711, 15.824265480041504, -4.734807014465332, -12.874336242675781, 2.30733585357666, 2.513456106185913, 7.456311225891113, -4.4685540199279785, -6.222180366516113, 7.3939032554626465, 12.059170722961426], [6.925175666809082, 1.0354101657867432, -0.20873917639255524, 4.031195163726807, 3.569908380508423, 10.954344749450684, -4.041675567626953, 9.339287757873535, 2.216581106185913, 9.566080093383789, 1.4963319301605225, -5.166959285736084, -7.98341703414917, -10.603193283081055, -1.5508754253387451, -6.062282085418701, 19.533557891845703, 9.499140739440918, -1.7954862117767334, -4.2738471031188965, 13.454957962036133, 12.746684074401855, 3.786707878112793, -2.395965576171875, -6.4927825927734375, 14.113767623901367, 2.4677834510803223, 1.7467764616012573, -6.853559494018555, -2.3767850399017334, -11.069811820983887, 4.047961235046387, 0.8641011118888855], [8.869108200073242, 4.1792449951171875, 6.06489896774292, -6.455156326293945, 18.41313362121582, -4.170543193817139, 8.054768562316895, 3.2675275802612305, 1.6609548330307007, 2.7230751514434814, 9.098941802978516, -3.4914298057556152, 2.6881513595581055, -13.011960983276367, -5.986324310302734, 8.192835807800293, 6.095054626464844, -0.8740628361701965, -4.441873550415039, -4.976719856262207, 2.3298568725585938, -2.8117220401763916, 7.500940322875977, 6.785107135772705, 2.8868415355682373, -13.762742042541504, 4.402615070343018, 3.888101100921631, 2.2487096786499023, -5.454874515533447, 8.557597160339355, 7.970586776733398, -2.5176658630371094], [1.1102582216262817, -5.935730934143066, 3.1928937435150146, 5.03113317489624, -7.560178279876709, -0.4894363284111023, -7.282978534698486, 4.249674320220947, 14.383787155151367, -4.66383171081543, 8.300825119018555, -2.5125882625579834, -4.775432586669922, -1.3333966732025146, 2.6261050701141357, -4.167637825012207, 6.096126079559326, 2.640052318572998, -5.911907196044922, 6.453078269958496, -3.7417235374450684, 10.50498104095459, -9.95705795288086, -8.597028732299805, -5.6957478523254395, 12.154963493347168, 3.369823932647705, 2.8855834007263184, -11.109221458435059, 1.103779911994934, -0.8450728058815002, 4.595043659210205, 8.618409156799316], [-1.989789605140686, 13.70544719696045, -13.27670669555664, -4.603444576263428, 1.1579176187515259, -6.408985614776611, 1.2868759632110596, -11.722696304321289, 9.114148139953613, -9.566643714904785, -0.9959458112716675, 4.177851676940918, 5.2532782554626465, -1.5590389966964722, 0.9270361065864563, 7.854371070861816, -11.712050437927246, -3.7247281074523926, 1.8740882873535156, -3.9056549072265625, -1.0301059484481812, -10.477974891662598, 17.388519287109375, -0.21162927150726318, 0.9914857149124146, -3.9979352951049805, 3.422100305557251, -6.023390293121338, -2.2407543659210205, 0.2488935887813568, -8.374783515930176, 17.404037475585938, 2.603480815887451]]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(split_k_matmul, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")